# Phi-4-mini-instruct / Numina to Omni-MATH: resume all three seeds

**Seeds: 42, 43, 44.** Two arms per seed: GRPO + PI (monitor), then GRPO + PI + NAFC.
This notebook embeds the saved preparation and every experiment source. No ZIP selection, repository access,
intermediate file transfer or other notebook is needed. Select **A100-SXM4-40GB**, then **Run all**.
Phi is public; no token or login is required.

Saved validation progress: **32/64 questions** (128 responses). Preparation and training are not complete yet.
The two old Phi ZIPs contain identical progress; one copy is embedded, without double-counting responses.
The saved model/configuration/source fingerprints are preserved. Software and GPU identity must still match.

Both new notebooks can run independently at the same time. Phi prepares once for all three training seeds,
saving the duplicated preparation of the previous two-Phi-notebook layout. It still runs six arms sequentially;
fewer concurrent GPUs do not halve the total training work. Completion within 500 CU or one session is unmeasured.

| Setting | Preserved protocol |
|---|---|
| Train / validation / test questions | 2,000 / 64 / 2,000 |
| Steps / responses | 300 steps per arm; 8 prompts x 8 fresh responses per step |
| Anchors | Exactly 128; 64 repair and 64 held controller |
| Selection / calibration | >=6/8 initial successes; three independent four-response noise audits |
| Online audits | Four responses on each anchor every 25 steps |
| Primary tests | Steps 50, 100, 150, 200, 300; four responses/question |
| Primary / secondary cap | 4096 / 8192 tokens; secondary at step 300 only |
| Reference | Eight responses plus independent four-response noise check at both caps |
| Training | LoRA r32, all linear layers, lr 1e-5, fp16 |

PI/NAFC formulas, data and evaluation budgets are unchanged. All milestones and capped/unboxed outputs stay in the report.
Omni-MATH uses the frozen numeric-answer subset, not the complete official benchmark or exclusively IMO problems.

Output remains local. At the end, click **Download all output (.zip)**. The archive includes results and full resume
checkpoints, but no model cache or credentials. A destroyed runtime loses progress that has not been downloaded.


## 2. Runtime setup

The saved CUDA PyTorch build must match; other recorded libraries are installed at their saved versions.
Do not replace the embedded sources or change the configuration of this resumed run.


In [ ]:
import os, sys, subprocess, time
import importlib.metadata
from pathlib import Path
ROOT = Path("/content/nafc_a100")
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["USE_TF"] = "0"
os.environ["HF_HOME"] = "/content/hf_cache"
EXPECTED_VERSIONS = {'torch': '2.11.0+cu130', 'transformers': '4.52.4', 'peft': '0.15.2', 'math-verify': '0.8.0', 'accelerate': '1.8.1', 'numpy': '2.1.3', 'matplotlib': '3.10.0'}
if importlib.metadata.version("torch") != EXPECTED_VERSIONS["torch"]:
    raise RuntimeError("This saved run needs PyTorch " + EXPECTED_VERSIONS["torch"] + ". Select the matching Colab runtime; do not bypass the saved experiment identity.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *['transformers==4.52.4', 'peft==0.15.2', 'math-verify==0.8.0', 'accelerate==1.8.1', 'numpy==2.1.3', 'matplotlib==3.10.0']], check=True)
ASSIGNED_SEEDS = [42, 43, 44]
SLUG = 'phi_olympic'
OUTPUT = Path("/content/results") / (SLUG + "_resume_3seeds")
OUTPUT.mkdir(parents=True, exist_ok=True)
ARCHIVES = Path("/content/nafc_archives")
ARCHIVES.mkdir(parents=True, exist_ok=True)


## 3. Experiment sources and saved preparation

### `core.py`

In [ ]:
%%writefile /content/nafc_a100/core.py
"""Repository PI/NAFC formulas; extracted unchanged from rl-nafc commit 3c18f8d."""

from __future__ import annotations

import math, json, re

from dataclasses import dataclass, field

def clipped_cost(length: int, budget: float) -> float:
    """Chi phí bị cắt của LACONIC: bằng 0 nếu trong ngân sách."""
    return max((length - budget) / budget, 0.0)

@dataclass
class LagrangianController:
    method: str
    budgets: list[float]            # B_k; với phương pháp toàn cục chỉ dùng budgets[0] = B
    max_len: int                    # L_max = max_completion_length
    difficulty: list[float] | None = None   # d_k trong [0,1], 0 = dễ, 1 = khó (dalg_param)
    eta: float = 0.002              # bước cập nhật đối ngẫu
    eta_param: float = 0.002        # bước cập nhật cho (a, b) của dalg_param
    fixed_lambda: float = 0.01
    optimistic: bool = False
    init_lambdas: list[float] | None = None
    leash_lambda_max: float = 1.0
    two_sided: bool = False         # λ được âm: ràng buộc coi là chủ động (E[L] = B), chống ép quá tay
    kp: float = 0.0                 # hệ số tỉ lệ (PI): λ = I + kp * vi_phạm, phản ứng ngay khi độ dài vượt/thiếu

    lambdas: list[float] = field(init=False)
    _prev_grad: list[float] = field(init=False)
    _a: float = field(init=False, default=0.0)
    _b: float = field(init=False, default=0.0)
    _prev_ab_grad: tuple[float, float] = field(init=False, default=(0.0, 0.0))
    _integral: list[float] = field(init=False)

    def __post_init__(self):
        valid = {"grpo", "fixed", "leash", "laconic", "dalg", "dalg_param"}
        if self.method not in valid:
            raise ValueError(f"method phải thuộc {valid}")
        k = self.num_groups
        if self.init_lambdas is not None:
            self.lambdas = [min(max(l, 0.0), self.ceiling(i)) for i, l in enumerate(self.init_lambdas[:k])]
        elif self.method == "fixed":
            self.lambdas = [self.fixed_lambda] * k
        else:
            self.lambdas = [0.0] * k
        self._prev_grad = [0.0] * k
        self._integral = list(self.lambdas)
        if self.method == "dalg_param":
            if self.difficulty is None or len(self.difficulty) != len(self.budgets):
                raise ValueError("dalg_param cần difficulty cùng độ dài với budgets")
            # Khởi tạo (a, b): đường thẳng qua lambda của nhóm dễ nhất và khó nhất (nếu có init), ngược lại (0, 0)
            if self.init_lambdas is not None:
                d0, d1 = self.difficulty[0], self.difficulty[-1]
                y0, y1 = self.lambdas[0], self.lambdas[-1]
                self._b = (y1 - y0) / (d1 - d0) if d1 != d0 else 0.0
                self._a = y0 - self._b * d0
            else:
                self._a, self._b = 0.0, 0.0
            self._sync_param_lambdas()

    # ------------------------------------------------------------------ #
    def state_dict(self) -> dict:
        """Lưu trạng thái đối ngẫu để tiếp tục train khi Colab bị ngắt."""
        return {"lambdas": self.lambdas, "prev_grad": self._prev_grad,
                "a": self._a, "b": self._b, "prev_ab_grad": list(self._prev_ab_grad),
                "integral": list(self._integral)}

    def load_state_dict(self, d: dict) -> None:
        self.lambdas = list(d["lambdas"])
        self._prev_grad = list(d["prev_grad"])
        self._a, self._b = d["a"], d["b"]
        self._prev_ab_grad = tuple(d["prev_ab_grad"])
        self._integral = list(d.get("integral", self.lambdas))

    @property
    def num_groups(self) -> int:
        return len(self.budgets) if self.method in ("dalg", "dalg_param") else 1

    def group_of(self, bucket: int) -> int:
        return bucket if self.method in ("dalg", "dalg_param") else 0

    def ceiling(self, k: int) -> float:
        """Trần của lambda. LACONIC: Lambda = B/(Lmax - B) để lời giải đúng luôn có reward > 0."""
        if self.method == "leash":
            return self.leash_lambda_max
        b = self.budgets[k] if self.method in ("dalg", "dalg_param") else self.budgets[0]
        return b / max(self.max_len - b, 1.0)

    def floor(self, k: int) -> float:
        """Sàn của λ: 0 cho ràng buộc bất đẳng thức; -trần khi two_sided."""
        return -self.ceiling(k) if self.two_sided else 0.0

    def budget(self, k: int) -> float:
        return self.budgets[k] if self.method in ("dalg", "dalg_param") else self.budgets[0]

    def _sync_param_lambdas(self):
        self.lambdas = [
            min(max(self._a + self._b * d, 0.0), self.ceiling(i))
            for i, d in enumerate(self.difficulty)
        ]

    # ------------------------------------------------------------------ #
    def shaped_rewards(self, correct: list[bool], lengths: list[int], buckets: list[int]) -> list[float]:
        """Primal: reward Lagrange cho từng completion (dùng lambda hiện tại)."""
        out = []
        for ok, L, bk in zip(correct, lengths, buckets):
            k = self.group_of(bk)
            B = self.budget(k)
            lam = self.lambdas[k]
            if self.method == "grpo":
                out.append(1.0 if ok else 0.0)
            elif self.method == "leash":
                r = 1.0 if ok else -1.0
                out.append(min(max(r - lam * clipped_cost(L, B), -1.0), 1.0))
            else:  # fixed, laconic, dalg, dalg_param
                r = 1.0 if ok else 0.0
                out.append(r - lam * clipped_cost(L, B))
        return out

    def dual_update(self, lengths: list[int], buckets: list[int]) -> dict[str, float]:
        """Dual: cập nhật lambda từ độ dài trung bình (dùng chi phí tuyến tính L/B - 1, như LACONIC/Leash)."""
        stats: dict[str, float] = {}
        k_total = self.num_groups
        sums = [0.0] * k_total
        counts = [0] * k_total
        for L, bk in zip(lengths, buckets):
            k = self.group_of(bk)
            sums[k] += L
            counts[k] += 1

        grads = [0.0] * k_total
        for k in range(k_total):
            if counts[k] == 0:
                continue
            mean_len = sums[k] / counts[k]
            grads[k] = mean_len / self.budget(k) - 1.0
            stats[f"constraint/mean_len_g{k}"] = mean_len
            stats[f"constraint/violation_g{k}"] = grads[k]

        if self.method in ("grpo", "fixed"):
            pass
        elif self.method == "dalg_param":
            # Gradient ascent trên (a, b): dlambda_k/da = 1, dlambda_k/db = d_k (khi lambda_k không bị cắt).
            # Mỗi nhóm đóng góp theo tỷ lệ số mẫu -> (a, b) học từ TOÀN BỘ batch, ít nhiễu hơn K lambda độc lập.
            n = sum(counts) or 1
            ga = gb = 0.0
            for k in range(k_total):
                if counts[k] == 0:
                    continue
                w = counts[k] / n
                ga += w * grads[k]
                gb += w * grads[k] * self.difficulty[k]
            if self.optimistic:
                pa, pb = self._prev_ab_grad
                step_a, step_b = 2 * ga - pa, 2 * gb - pb
            else:
                step_a, step_b = ga, gb
            self._prev_ab_grad = (ga, gb)
            self._a += self.eta_param * step_a
            self._b += self.eta_param * step_b
            self._sync_param_lambdas()
            stats["constraint/param_a"] = self._a
            stats["constraint/param_b"] = self._b
        else:  # leash, laconic, dalg
            for k in range(k_total):
                if counts[k] == 0:
                    continue
                g = 2 * grads[k] - self._prev_grad[k] if self.optimistic else grads[k]
                self._prev_grad[k] = grads[k]
                lo, hi = self.floor(k), self.ceiling(k)
                if self.kp > 0:   # PI: phần tích phân (có chống tích luỹ quá trần) + phần tỉ lệ
                    self._integral[k] = min(max(self._integral[k] + self.eta * g, lo), hi)
                    self.lambdas[k] = min(max(self._integral[k] + self.kp * grads[k], lo), hi)
                else:
                    self.lambdas[k] = min(max(self.lambdas[k] + self.eta * g, lo), hi)

        for k, lam in enumerate(self.lambdas):
            stats[f"constraint/lambda_g{k}"] = lam
        return stats

MODES = ("nafc", "flipguard", "robust_fixed", "raw_adaptive", "monitor", "nafc_replay", "replay_fixed")

COUNTING = {"nafc": "robust", "flipguard": "raw", "robust_fixed": "robust", "raw_adaptive": "raw", "monitor": "robust",
            "nafc_replay": "robust", "replay_fixed": "robust"}

ADAPTIVE = {"nafc": True, "flipguard": False, "robust_fixed": False, "raw_adaptive": True, "monitor": False,
            "nafc_replay": True, "replay_fixed": False}

PULL = {"nafc": "nll", "flipguard": "nll", "robust_fixed": "nll", "raw_adaptive": "nll", "monitor": "none",
        "nafc_replay": "replay", "replay_fixed": "replay"}

def split_anchors(n: int, frac: float, seed: int = 0) -> tuple[list, list]:
    """Chia cố định n anchor: round(frac·n) anchor SỬA (được train), phần còn lại anchor ĐO (chỉ đo)."""
    import random
    perm = list(range(n))
    random.Random(seed).shuffle(perm)
    k = int(round(frac * n))
    return sorted(perm[:k]), sorted(perm[k:])

def n_replay_groups(lam: float, frac_max: float, n_groups: int, pool_size: int) -> int:
    """Số nhóm prompt thay bằng anchor: ceil(λ · frac_max · G), không vượt số anchor trong pool
    và luôn chừa ít nhất 1 prompt train."""
    if lam <= 0 or pool_size <= 0 or n_groups <= 1:
        return 0
    n = math.ceil(lam * frac_max * n_groups - 1e-9)
    return max(0, min(n, pool_size, n_groups - 1))

def replace_groups(inputs: list, n_replace: int, pool: list, ptr: int, make_item, key=None):
    """Thay n_replace nhóm prompt ĐẦU TIÊN của batch (dataset đã xáo) bằng anchor lấy lần lượt từ pool.
    Mọi phần tử cùng prompt (cùng nhóm GRPO) được thay bằng cùng một anchor.
    make_item(x, anchor_idx) -> phần tử mới. Trả về (inputs_mới, ptr_mới, danh_sách_anchor_đã_dùng)."""
    key = key or (lambda x: json.dumps(x["prompt"], sort_keys=True, ensure_ascii=False))
    order = []
    for x in inputs:
        k = key(x)
        if k not in order:
            order.append(k)
    chosen, used = {}, []
    for k in order[:n_replace]:
        a = pool[ptr % len(pool)]
        ptr += 1
        chosen[k] = a
        used.append(a)
    out = [make_item(x, chosen[key(x)]) if key(x) in chosen else x for x in inputs]
    return out, ptr, used

def robust_flags(correct: list[list[bool]], lo: float = 0.25) -> list[bool]:
    """Câu hỏng thật: tỷ lệ đúng của model hiện tại trên k mẫu ≤ lo (anchor đã chọn sẵn p^_gốc ≥ 0,75)."""
    return [len(c) > 0 and sum(c) / len(c) <= lo for c in correct]

def raw_flags(correct: list[list[bool]], base_first_correct: list[bool]) -> list[bool]:
    """Câu hỏng thô (kiểu FlipGuard): MỘT mẫu gốc đúng và MỘT mẫu mới sai (mẫu đầu tiên của mỗi bên)."""
    return [bool(b) and len(c) > 0 and not c[0] for c, b in zip(correct, base_first_correct)]

@dataclass
class FlipController:
    mode: str = "nafc"
    eps: float = 0.06            # ngân sách: tỷ lệ anchor bị hỏng cho phép (cùng cách đếm với chế độ)
    eta: float = 1.0             # bước đối ngẫu
    gamma: float = 0.1           # λ cố định cho flipguard / robust_fixed
    lam_max: float = 1.0
    lam: float = 0.0
    selected: list = field(default_factory=list)   # chỉ số anchor trong S (lần kiểm tra gần nhất)
    history: list = field(default_factory=list)
    repair_idx: list | None = None   # tách anchor: anchor SỬA (None = không tách, mọi anchor vừa sửa vừa đo)
    held_idx: list | None = None     # tách anchor: anchor ĐO, λ chạy theo F^ trên tập này

    @property
    def split(self) -> bool:
        return self.held_idx is not None

    def __post_init__(self):
        if self.mode not in MODES:
            raise ValueError(f"mode phải thuộc {MODES}, nhận '{self.mode}'")
        if self.mode in ("flipguard", "robust_fixed", "replay_fixed"):
            self.lam = self.gamma
        if self.mode == "monitor":
            self.lam = 0.0

    @property
    def counting(self) -> str:
        return COUNTING[self.mode]

    @property
    def adaptive(self) -> bool:
        return ADAPTIVE[self.mode]

    @property
    def pull_kind(self) -> str:
        return PULL[self.mode]

    def replay_pool(self, n_anchor: int) -> list:
        """nafc_replay: anchor hỏng ở lần kiểm tra gần nhất (trong nửa SỬA nếu tách); replay_fixed: mọi anchor SỬA."""
        if self.mode == "replay_fixed":
            return list(self.repair_idx) if self.split else list(range(n_anchor))
        return list(self.selected)

    def update(self, robust: list[bool], raw: list[bool], step: int = -1, extra: dict | None = None) -> dict:
        """Một lần kiểm tra anchor: chọn S theo cách đếm của chế độ, cập nhật λ, trả về số liệu để ghi log."""
        n = max(len(robust), 1)
        f_robust, f_raw = sum(robust) / n, sum(raw) / n
        flags = robust if self.counting == "robust" else raw
        f = f_robust if self.counting == "robust" else f_raw
        self.selected = [i for i, x in enumerate(flags) if x]
        split_rec = {}
        if self.split:
            frac = lambda fl, idx: sum(1 for i in idx if fl[i]) / max(len(idx), 1)
            split_rec = {"F_robust_held": frac(robust, self.held_idx), "F_robust_repair": frac(robust, self.repair_idx),
                         "F_raw_held": frac(raw, self.held_idx), "F_raw_repair": frac(raw, self.repair_idx),
                         "n_held": len(self.held_idx), "n_repair": len(self.repair_idx)}
            f = frac(flags, self.held_idx)                                        # λ theo nửa ĐO
            rep_set = set(self.repair_idx)
            self.selected = [i for i in self.selected if i in rep_set]            # chỉ sửa nửa SỬA
        lam_before = self.lam
        if self.mode == "monitor":
            self.lam = 0.0
        elif self.adaptive:
            self.lam = min(max(self.lam + self.eta * (f - self.eps), 0.0), self.lam_max)
        else:
            self.lam = self.gamma
        rec = {"step": step, "mode": self.mode, "F_robust": f_robust, "F_raw": f_raw, "F_used": f, "eps": self.eps,
               "lam_before": lam_before, "lam": self.lam, "n_S": len(self.selected), "n_anchor": len(robust)}
        rec.update(split_rec)
        if extra:
            rec.update(extra)
        self.history.append(rec)
        return rec

    @property
    def pulling(self) -> bool:
        if self.mode == "replay_fixed":
            return self.lam > 0
        return self.lam > 0 and bool(self.selected)

    def state_dict(self) -> dict:
        return {"mode": self.mode, "eps": self.eps, "eta": self.eta, "gamma": self.gamma, "lam_max": self.lam_max,
                "lam": self.lam, "selected": list(self.selected), "history": list(self.history),
                "repair_idx": self.repair_idx, "held_idx": self.held_idx}

    def load_state_dict(self, d: dict) -> None:
        if d.get("mode") != self.mode:
            raise ValueError(f"checkpoint dùng chế độ {d.get('mode')}, lần chạy này là {self.mode}")
        if d.get("held_idx") != self.held_idx:
            raise ValueError("checkpoint dùng cách tách anchor khác lần chạy này (--anchor_split / --split_seed)")
        self.lam, self.selected, self.history = float(d["lam"]), list(d["selected"]), list(d["history"])

def default_eps(meta: dict, mode: str, margin: float = 0.02) -> float:
    """ε = mức nhiễu của CHÍNH cách đếm mà chế độ dùng (model gốc chấm lại trên anchor) + biên."""
    key = "noise_robust" if COUNTING[mode] == "robust" else "noise_raw"
    if key not in meta:
        raise KeyError(f"file anchor thiếu '{key}', chạy lại build_anchors.py")
    return float(meta[key]) + margin

def decoupled_advantages(group_keys, rewards, costs, lams, eps=1e-6):
    groups = {}
    for i, g in enumerate(group_keys):
        groups.setdefault(g, []).append(i)
    out = [0.0] * len(rewards)
    for idx in groups.values():
        def z(vals):
            m = sum(vals) / len(vals)
            s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
            return [0.0 if s < eps else (v - m) / s for v in vals]
        zr = z([rewards[i] for i in idx])
        zc = z([costs[i] for i in idx])
        for j, i in enumerate(idx):
            out[i] = zr[j] - lams[i] * zc[j]
    return out

SYSTEM_PROMPT = "Please reason step by step, and put your final answer within \\boxed{}."

def gold_answer(dataset: str, raw: str) -> str:
    if dataset == "gsm8k":
        return raw.split("####")[-1].strip().replace(",", "")
    if dataset == "math":
        return (last_boxed(raw) or "").strip()
    return raw.strip()

def chat_messages(question: str) -> list[dict]:
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

def last_boxed(text: str) -> str | None:
    """Nội dung của \\boxed{...} cuối cùng, xử lý ngoặc lồng nhau."""
    idx = text.rfind("\\boxed")
    if idx < 0:
        return None
    i = text.find("{", idx)
    if i < 0:
        return None
    depth, j = 0, i
    while j < len(text):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i + 1 : j]
        j += 1
    return None

def _normalize(s: str) -> str:
    s = s.strip().strip("$").replace(" ", "").replace("\\!", "").replace(",", "")
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac").replace("\\left", "").replace("\\right", "")
    s = s.rstrip(".")
    return s

def _as_number(s: str) -> float | None:
    s = _normalize(s)
    m = re.fullmatch(r"-?\d+(\.\d+)?", s)
    if m:
        return float(s)
    m = re.fullmatch(r"(-?)\\frac\{(-?\d+)\}\{(\d+)\}", s)
    if m and int(m.group(3)) != 0:
        sign = -1 if m.group(1) else 1
        return sign * int(m.group(2)) / int(m.group(3))
    return None

def is_correct(completion: str, gold: str) -> bool:
    """Chấm đúng/sai. Ưu tiên math_verify nếu đã cài, ngược lại so sánh chuẩn hoá."""
    pred = last_boxed(completion)
    if pred is None:
        return False
    try:
        from math_verify import parse, verify

        g, p = parse(f"${gold}$"), parse(f"${pred}$")
        if g and p:
            return bool(verify(g, p))
    except Exception:
        pass
    a, b = _as_number(pred), _as_number(gold)
    if a is not None and b is not None:
        return abs(a - b) < 1e-6 * max(1.0, abs(b))
    return _normalize(pred) == _normalize(gold)


### `SOURCE.json`

In [ ]:
%%writefile /content/nafc_a100/SOURCE.json
{
  "upstream_commit": "3c18f8dfffbe9a0ac0f95533a0e971cc3687be51",
  "files": {
    "constraints.py": {
      "sha256": "b6faa0ad66e063e06603aa75e9780f67c4518edbba7a24f8723dc39fdc65bc43",
      "symbols": [
        "clipped_cost",
        "LagrangianController"
      ]
    },
    "flipcon.py": {
      "sha256": "6658769eef936200f557272948e3c7d3c7bf7171907be9bd4d04669195a54875",
      "symbols": [
        "MODES",
        "COUNTING",
        "ADAPTIVE",
        "PULL",
        "split_anchors",
        "n_replay_groups",
        "replace_groups",
        "robust_flags",
        "raw_flags",
        "FlipController",
        "default_eps"
      ]
    },
    "train.py": {
      "sha256": "a41a6d510d8819144d0cd29d0faa573ecb58ae25a47a9a1760cd4ffb0dac696c",
      "symbols": [
        "decoupled_advantages"
      ]
    },
    "common.py": {
      "sha256": "a4950e6ca79368f1f741ccc3c3c918272754c80751afd59452764a97badf68b3",
      "symbols": [
        "SYSTEM_PROMPT",
        "gold_answer",
        "chat_messages",
        "last_boxed",
        "_normalize",
        "_as_number",
        "is_correct"
      ]
    }
  },
  "core_sha256": "c5691fe4c62585df5573c6af26f1448fa6d40cdeb64fde87c7dbe2233a98111e",
  "core_line_endings": "LF; symbols identical to upstream AST"
}


### `io_utils.py`

In [ ]:
%%writefile /content/nafc_a100/io_utils.py
"""Small file helpers and immutable run identity."""
import hashlib
import json
import os
import tempfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent
OUT = ROOT / "outputs"


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name + ".", suffix=".tmp", delete=False) as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    try:
        os.replace(f.name, path)
    finally:
        if Path(f.name).exists():
            Path(f.name).unlink()


def freeze(path, value):
    path = Path(path)
    if path.exists():
        if read(path) != value:
            raise ValueError(f"Different frozen experiment: {path}. Use a fresh outputs directory.")
    else:
        write(path, value)


def checkpoint_identity(folder):
    folder = Path(folder).resolve()
    if not folder.is_dir():
        raise ValueError("Reference checkpoint is missing; run shared preparation first.")
    config_name = "config.json"
    weights = sorted(folder.glob("*.safetensors"))
    if not weights or not (folder / config_name).exists():
        raise ValueError("Reference needs config.json and safetensors weights.")
    c = read(folder / config_name)
    if c.get("model_type") not in ("phi3", "gemma3_text"):
        raise ValueError("This experiment requires a pinned Phi or Gemma checkpoint.")
    files = weights + [folder / config_name]
    files += [p for p in folder.iterdir() if p.is_file() and
              ("tokenizer" in p.name or p.name in ("vocab.json", "merges.txt", "special_tokens_map.json", "generation_config.json")
               or p.name.endswith("safetensors.index.json"))]
    return {"kind": "merged", "files_sha256": {p.name: file_hash(p) for p in files}}


def validate_config(c):
    if c["base_model"] != "microsoft/Phi-4-mini-instruct":
        raise ValueError("Do not change the backbone inside a frozen pilot.")
    for k in ("steps", "prompts_per_step", "group_size", "train_count", "validation_count", "test_count",
              "anchor_count", "anchor_pool", "anchor_selection_batch", "initial_draws", "eval_draws", "generation_batch", "audit_every",
              "warmup_steps", "max_new_tokens", "max_prompt_tokens", "lora_rank", "long_eval_tokens"):
        if not isinstance(c[k], int) or c[k] <= 0:
            raise ValueError(f"{k} must be a positive integer")
    if c["group_size"] < 2 or c["initial_draws"] != 8 or c["eval_draws"] != 4:
        raise ValueError("The frozen protocol uses group size >=2, 8 initial draws and 4 evaluation draws.")
    if c["anchor_count"] % 2 or c["anchor_pool"] < c["anchor_count"]:
        raise ValueError("Anchors must split equally into repair/control halves.")
    if c["train_count"] % c["prompts_per_step"]:
        raise ValueError("train_count must be divisible by prompts_per_step to preserve complete prompt groups.")
    if not 0 < c["compression_ratio"] < 1 or c["precision"] not in ("fp16", "bf16"):
        raise ValueError("Invalid compression ratio or precision")
    if any(s <= 0 or s >= c["steps"] for s in c["validation_steps"]):
        raise ValueError("Validation milestones must precede the final step.")
    if not 0 <= c["noise_margin"] < 1 or not 0 < c["replay_frac_max"] < 1:
        raise ValueError("Invalid noise margin or replay fraction")
    if not 0 < c["fixed_replay_fraction"] <= c["replay_frac_max"]:
        raise ValueError("Fixed replay must use the same maximum replacement budget.")
    if c["long_eval_tokens"] <= c["max_new_tokens"]:
        raise ValueError("Long evaluation must exceed the primary token cap.")
    if len(set(c["seeds"])) != len(c["seeds"]) or len(c["seeds"]) < 2 or c["seed"] not in c["seeds"]:
        raise ValueError("Use the prespecified independent training seeds.")
    for k in ("learning_rate", "pi_eta", "pi_kp", "pi_lambda_max", "nafc_eta", "nafc_lambda_max"):
        if not isinstance(c[k], (int, float)) or not 0 < c[k] < float("inf"):
            raise ValueError(f"Invalid {k}")


def experiment_identity(c, parent, splits, environment):
    settings = {k: v for k, v in c.items() if k != "reference_checkpoint"}
    code = {p.name: file_hash(p) for p in sorted(ROOT.glob("*.py"))}
    return {"protocol": "phi-olympic-pi-nafc-v1", "settings": settings,
            "reference": parent, "splits_sha256": digest(splits), "code_sha256": code,
            "source": read(ROOT / "SOURCE.json"), "environment": environment}


### `scoring.py`

In [ ]:
%%writefile /content/nafc_a100/scoring.py
"""Numeric-answer Math-Verify scoring, with bounded Windows worker calls."""
import atexit
import json
import os
import queue
import re
import subprocess
import sys
import threading
from functools import lru_cache
from pathlib import Path
from math_verify import parse, verify
from core import last_boxed, _normalize

_worker = None


def _parse_numeric(answer, timeout):
    if not answer or len(answer) > 256:
        return None
    # Reject prose, symbolic variables and multiple answers before any extraction.
    plain = re.sub(r"\\(?:dfrac|tfrac|frac|sqrt|pi|cdot|times|left|right|boxed)\b", "", answer)
    if re.search(r"[A-Za-z,;=<>]", plain) or re.search(r"[^0-9\s.${}()\[\]+*/^!\-\\]", plain):
        return None
    parsed = parse(f"${answer.strip('$')}$", fallback_mode="no_fallback", parsing_timeout=timeout)
    value = next((x for x in parsed if not isinstance(x, str)), None)
    return value if getattr(value, "is_number", None) is True and getattr(value, "is_finite", None) is True else None


def _compute(request, timeout):
    if request['op'] == 'verify_math':
        if _normalize(request['gold']) == _normalize(request['pred']):
            return True
        gold = parse(f"${request['gold']}$", fallback_mode='no_fallback', parsing_timeout=timeout)
        pred = parse(f"${request['pred']}$", fallback_mode='no_fallback', parsing_timeout=timeout)
        return bool(gold and pred and verify(gold, pred, timeout_seconds=timeout))
    gold = _parse_numeric(request["gold"], timeout)
    if request["op"] == "valid":
        return gold is not None
    pred = _parse_numeric(request["pred"], timeout)
    return gold is not None and pred is not None and bool(verify(gold, pred, timeout_seconds=timeout))


def _stop_worker():
    global _worker
    if _worker is not None:
        if _worker.poll() is None:
            _worker.kill()
        _worker.wait()
        _worker.stdin.close(); _worker.stdout.close()
        _worker = None


def _bounded(request):
    global _worker
    if os.name != "nt":
        return _compute(request, 5)
    # Math-Verify's Windows timeout spawns a non-picklable nested function.
    # A persistent plain subprocess avoids that bug; the parent enforces the deadline.
    if _worker is None:
        _worker = subprocess.Popen([sys.executable, str(Path(__file__).resolve()), "--worker"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            text=True, encoding="utf-8")
    inbox = queue.Queue()
    try:
        _worker.stdin.write(json.dumps(request)+"\n"); _worker.stdin.flush()
        threading.Thread(target=lambda: inbox.put(_worker.stdout.readline()), daemon=True).start()
        response = inbox.get(timeout=10)
        if not response:
            raise RuntimeError("Scoring worker exited")
        result = json.loads(response)
        if "error" in result:
            raise RuntimeError(result["error"])
        return result["result"]
    except Exception as exc:
        _stop_worker()
        raise RuntimeError("Answer verification failed or timed out; no false reward was assigned.") from exc


@lru_cache(maxsize=32768)
def numeric_gold(answer):
    return answer if _bounded({"op": "valid", "gold": answer}) else None


def is_correct(completion, gold):
    pred = last_boxed(completion)
    if pred is None or numeric_gold(gold) is None or numeric_gold(pred) is None:
        return False
    return _bounded({"op": "verify", "gold": gold, "pred": pred})


def math_is_correct(completion, gold):
    pred = last_boxed(completion)
    return pred is not None and bool(_bounded({'op': 'verify_math', 'gold': gold, 'pred': pred}))


def math_gold(solution):
    answer = last_boxed(solution)
    if answer is None:
        unbraced = re.findall(r'\\boxed\s+([+-]?\d+(?:\.\d+)?)(?=\s|\$|[.,]|$)', solution)
        answer = unbraced[-1] if unbraced else None
    return answer


atexit.register(_stop_worker)


if __name__ == "__main__":
    import logging
    logging.disable(logging.WARNING)
    for line in sys.stdin:
        try:
            result = {"result": _compute(json.loads(line), None)}
        except Exception as exc:
            result = {"error": str(exc)}
        print(json.dumps(result), flush=True)


### `data.py`

In [ ]:
%%writefile /content/nafc_a100/data.py
"""Frozen Olympic train/holdout partitions, independent of training seed."""
import hashlib
import random
import re
from collections import Counter
from scoring import numeric_gold

DATASETS = {"numina": "AI-MO/NuminaMath-1.5", "omni": "KbsdJames/Omni-MATH"}


def normalized(text):
    return re.sub(r"[\s$\\{}]+", "", text.lower())


def rows(ds, domain, split, sources=None):
    result, seen = [], set()
    for i, item in enumerate(ds):
        if domain == "numina":
            if item["source"] not in sources or item["question_type"] != "math-word-problem":
                continue
            if any(str(item.get(k, "")).strip().lower() != "yes"
                   for k in ("problem_is_valid", "solution_is_valid")):
                continue
        answer = str(item.get("answer", "")).strip()
        if numeric_gold(answer) is None:
            continue
        question = item["problem"].strip()
        key = normalized(question)
        if not key or key in seen:
            continue
        seen.add(key)
        topics = item.get("domain", item.get("problem_type", "unknown"))
        topic = "; ".join(topics) if isinstance(topics, list) else str(topics)
        result.append({"id": hashlib.sha256(f"{domain}/{split}/{key}".encode()).hexdigest(),
                       "domain": domain, "question": question, "answer": answer,
                       "topic": topic, "source": item.get("source", domain),
                       "source_index": i, "difficulty": item.get("difficulty")})
    return result


def take(items, n, seed):
    if len(items) < n:
        raise ValueError(f"Need {n} eligible Olympic questions, available {len(items)}. Counts are not silently reduced.")
    return random.Random(seed).sample(items, n)


def evaluation_domains(data):
    return tuple(data["provenance"].get("evaluation_domains", ("numina", "omni")))


def validate_splits(data, c):
    counts = {"task_train": c["train_count"], "numina_validation": c["validation_count"],
              "numina_test": c["test_count"], "anchor_candidates": c["anchor_pool"]}
    seen_ids, seen_questions = set(), set()
    for name in (*counts, "omni_test"):
        if name in counts and len(data[name]) != counts[name]:
            raise ValueError(f"Wrong frozen count for {name}")
        if not data[name]:
            raise ValueError(f"Empty split: {name}")
        for q in data[name]:
            key = normalized(q["question"])
            if q["id"] in seen_ids or key in seen_questions:
                raise ValueError(f"Question leakage or duplicate in {name}")
            if not q["answer"]:
                raise ValueError("Empty gold answer")
            seen_ids.add(q["id"]); seen_questions.add(key)


def prepare_splits(c):
    from datasets import load_dataset
    print("[data] Loading Olympic-labelled Numina sources and Omni-MATH", flush=True)
    omni_raw = load_dataset(DATASETS["omni"], revision=c["omni_revision"], split="test")
    # Block every benchmark question, even ones outside our numeric-answer scoring scope.
    blocked = {normalized(q["problem"]) for q in omni_raw}
    omni = rows(omni_raw, "omni", "test")
    raw = load_dataset(DATASETS["numina"], revision=c["numina_revision"], split="train")
    allowed = rows(raw, "numina", "research_pool", c["numina_sources"])
    pool = [q for q in allowed if normalized(q["question"]) not in blocked]
    a = c["train_count"]; b = a + c["validation_count"]; d = b + c["test_count"]
    picked = take(pool, d + c["anchor_pool"], c["split_seed"])
    result = {"task_train": picked[:a], "numina_validation": picked[a:b],
              "numina_test": picked[b:d], "anchor_candidates": picked[d:], "omni_test": omni,
              "provenance": {"task_domain": "numina", "datasets": DATASETS,
                  "revisions": {k: c[f"{k}_revision"] for k in DATASETS},
                  "numina_sources": c["numina_sources"], "eligible_before_dedup": len(allowed),
                  "eligible_after_omni_dedup": len(pool), "omni_total": len(omni_raw),
                  "omni_scored": len(omni),
                  "split_status": "Numina research holdout; Omni numeric-answer subset, not the full official benchmark score.",
                  "filters": "Olympiad-labelled Numina sources; valid word problems; scalar numeric expressions only; no success/difficulty filter.",
                  "deduplication": "Normalized exact matching against ALL Omni questions. Paraphrase overlap is not certified."}}
    validate_splits(result, c)
    return result


def schedule(items, steps, prompts, seed):
    if len(items) < prompts:
        raise ValueError("Not enough unique questions for one batch")
    batches, cycle = [], 0
    while len(batches) < steps:
        epoch = list(items)
        random.Random(seed + cycle).shuffle(epoch)
        batches.extend(epoch[i:i + prompts] for i in range(0, len(epoch) - prompts + 1, prompts))
        cycle += 1
    return batches[:steps]


def describe(data):
    return {k: {"count": len(v), "topics": dict(Counter(q["topic"] for q in v)),
                "sources": dict(Counter(q["source"] for q in v))}
            for k, v in data.items() if k != "provenance"}


### `model.py`

In [ ]:
%%writefile /content/nafc_a100/model.py
"""HF batched sampling and one-update clipped Dr. GRPO for Phi/Gemma LoRA."""
import gc
import inspect
import os
import time
from contextlib import contextmanager, nullcontext
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
import torch.nn.functional as F
import transformers.utils.import_utils as imports
imports._librosa_available = False
from core import chat_messages, last_boxed
from scoring import is_correct, math_is_correct
from io_utils import ROOT, OUT, read


def dtype(c):
    return {"fp16": torch.float16, "bf16": torch.bfloat16}[c["precision"]]


def amp(model):
    t = getattr(model, "_pilot_dtype", None)
    return torch.autocast("cuda", dtype=t) if t is not None else nullcontext()


@contextmanager
def isolated_rng(seed):
    with torch.random.fork_rng(devices=[torch.cuda.current_device()] if torch.cuda.is_available() else []):
        torch.manual_seed(seed)
        yield


def check_gpu(c):
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU and CUDA-enabled PyTorch are required.")
    if c["precision"] == "bf16" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("This GPU does not support bf16. Set fp16 before preparing a NEW experiment.")


def base_path(c):
    from huggingface_hub import snapshot_download
    return snapshot_download(c["base_model"], revision=c["base_revision"],
                             cache_dir=str(ROOT / "hf_cache" / "hub"),
                             allow_patterns=["*.json", "*.safetensors", "*.txt"])


def load_model(c, path, trainable=False):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    torch.manual_seed(c["seed"])
    tok = AutoTokenizer.from_pretrained(str(path))
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(str(path), torch_dtype=dtype(c),
        attn_implementation="sdpa", low_cpu_mem_usage=True, device_map={"": 0})
    if c['max_prompt_tokens'] + max(c['max_new_tokens'], c['long_eval_tokens']) > model.config.max_position_embeddings:
        raise ValueError('Configured prompt plus response exceeds the native model context.')
    if trainable:
        model = get_peft_model(model, LoraConfig(r=c["lora_rank"], lora_alpha=2 * c["lora_rank"],
              lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
        for p in model.parameters():
            if p.requires_grad:
                p.data = p.data.float()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        model.config.use_cache = False
    model._pilot_dtype = dtype(c)
    model._pilot_chunked_head = c.get("chunked_head", False)
    return model, tok


def prompt_ids(tok, question):
    return tok.apply_chat_template(chat_messages(question), tokenize=True, add_generation_prompt=True)


def token_logps(model, p, response):
    ids = torch.tensor([p + response], device=next(model.parameters()).device)
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    if getattr(model, "_pilot_chunked_head", False):
        from torch.utils.checkpoint import checkpoint
        with amp(model):
            hidden = base.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False,
                                return_dict=True).last_hidden_state[0, len(p)-1:-1]
        head = model.get_output_embeddings()
        def chunk_logps(h, labels):
            with amp(model):
                logits = head(h)
                softcap = getattr(base.config, 'final_logit_softcapping', None)
                if softcap is not None:
                    logits = torch.tanh(logits / softcap) * softcap
                return -F.cross_entropy(logits.float(), labels, reduction="none")
        # Recompute small vocabulary projections in backward instead of retaining full softmax tensors.
        return torch.cat([checkpoint(chunk_logps, hidden[i:i+32], ids[0, len(p)+i:len(p)+i+32],
                         use_reentrant=False) for i in range(0, len(response), 32)])
    extra = {"logits_to_keep": len(response) + 1} if "logits_to_keep" in inspect.signature(base.forward).parameters else {}
    with amp(model):
        logits = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False, **extra).logits
    logits = logits[0, -len(response) - 1:-1]
    targets = ids[0, -len(response):]
    return torch.cat([-F.cross_entropy(logits[i:i + 32].float(), targets[i:i + 32], reduction="none")
                      for i in range(0, len(response), 32)])


def clipped_loss(logps, old_logps, advantage, max_tokens):
    ratio = torch.exp(logps - old_logps)
    return -torch.minimum(ratio * advantage, ratio.clamp(0.8, 1.2) * advantage).sum() / max_tokens


class Sampler:
    def __init__(self, model, tok, c, scorer=is_correct):
        self.model, self.tok, self.c = model, tok, c
        self.scorer = math_is_correct if c.get('test_domain') == 'math' and scorer is is_correct else scorer

    @torch.no_grad()
    def __call__(self, problems, n, seed):
        expanded = [(q, j) for q in problems for j in range(n)]
        result = []
        flags = [(m, m.training) for m in self.model.modules()]
        self.model.eval()
        endings = self.model.generation_config.eos_token_id
        endings = [endings] if isinstance(endings, int) else list(endings or [self.tok.eos_token_id])
        started = time.monotonic()
        if self.c.get("sample_progress"):
            print(f"[sample] generating {len(expanded)} responses; sample={self.c.get('do_sample', True)}", flush=True)
        try:
            for offset in range(0, len(expanded), self.c["generation_batch"]):
                chunk = expanded[offset:offset + self.c["generation_batch"]]
                prompts = [prompt_ids(self.tok, q["question"]) for q, _ in chunk]
                if max(map(len, prompts)) > self.c["max_prompt_tokens"]:
                    raise ValueError("Prompt exceeds configured cap; no silent prompt truncation.")
                width = max(map(len, prompts))
                ids = torch.tensor([[self.tok.pad_token_id] * (width - len(p)) + p for p in prompts],
                                   device=next(self.model.parameters()).device)
                mask = torch.tensor([[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=ids.device)
                base = self.model.get_base_model() if hasattr(self.model, "get_base_model") else self.model
                kwargs = {"logits_to_keep": 1} if "logits_to_keep" in inspect.signature(base.forward).parameters else {}
                sampling = dict(do_sample=self.c.get("do_sample", True), temperature=1.0, top_p=1.0,
                                top_k=0 if self.c.get("do_sample", True) else 50)
                with isolated_rng(seed + offset * 1009), amp(self.model):
                    generated = self.model.generate(input_ids=ids, attention_mask=mask,
                        repetition_penalty=1.0, max_new_tokens=self.c["max_new_tokens"],
                        use_cache=True, pad_token_id=self.tok.pad_token_id, eos_token_id=endings, **sampling, **kwargs)
                for (q, j), generated_ids in zip(chunk, generated[:, width:].tolist()):
                    stop = next((i for i, token in enumerate(generated_ids) if token in endings), None)
                    ended = stop is not None
                    if ended:
                        generated_ids = generated_ids[:stop + 1]
                    text = self.tok.decode(generated_ids, skip_special_tokens=True)
                    result.append({"id": q["id"], "sample_index": j, "tokens": generated_ids,
                                   "text": text, "correct": bool(self.scorer(text, q["answer"])),
                                   "capped": not ended and len(generated_ids) >= self.c["max_new_tokens"],
                                   "at_limit": len(generated_ids) >= self.c["max_new_tokens"],
                                   "no_box": last_boxed(text) is None})
                if self.c.get("sample_progress") and (len(result) % 16 == 0 or len(result) == len(expanded)):
                    print(f"[sample] {len(result)}/{len(expanded)} ({time.monotonic()-started:.0f}s)", flush=True)
        finally:
            for module, training in flags:
                module.training = training
        return result


def evaluate(sampler, problems, n, seed, path):
    from io_utils import digest, write
    identity = {"questions": digest(problems), "draws": n, "seed": seed,
                "sampling": getattr(sampler, "c", None)}
    state = read(path) if Path(path).exists() else {"identity": identity, "records": []}
    if state["identity"] != identity:
        raise ValueError(f"Evaluation identity mismatch: {path}")
    done = len(state["records"]) // n
    expected = [q["id"] for q in problems[:done] for _ in range(n)]
    if [r["id"] for r in state["records"]] != expected or any(
            r["sample_index"] != i % n for i, r in enumerate(state["records"])):
        raise ValueError("Incomplete or reordered saved evaluation")
    for start in range(done, len(problems), 16):
        chunk = problems[start:start + 16]
        new = sampler(chunk, n, seed + start * n * 1009)
        if [(r["id"], r["sample_index"]) for r in new] != [(q["id"], j) for q in chunk for j in range(n)]:
            raise ValueError("Incomplete or reordered new evaluation")
        state["records"].extend(new)
        write(path, state)
        print(f"[eval {Path(path).stem}] {min(start + 16, len(problems))}/{len(problems)}", flush=True)
    return state["records"]


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### `engine.py`

In [ ]:
%%writefile /content/nafc_a100/engine.py
"""One optimizer step per rollout batch; the repository's PI and NAFC on-policy rules."""
import os
import math
import time
from pathlib import Path
import torch
from core import LagrangianController, FlipController, decoupled_advantages, n_replay_groups, replace_groups, robust_flags, raw_flags
from data import schedule, evaluation_domains
from io_utils import OUT, read, write, digest
from model import load_model, Sampler, prompt_ids, token_logps, clipped_loss, evaluate

ARMS = ("grpo_pi", "grpo_pi_fixed_replay", "grpo_pi_nafc")
PAIRED_ARMS = ("grpo_pi", "grpo_pi_nafc")


def test_path(folder, step, final_step, domain):
    name = 'final' if step == final_step else f'step_{step}'
    return folder/f'{name}_{domain}_test.json'


def audit(sampler, bundle, ctrl, c, step):
    records = sampler(bundle["anchors"], 4, 500_000_000 + step * 1_000_000)
    correct = [[r["correct"] for r in records[i * 4:(i + 1) * 4]] for i in range(len(bundle["anchors"]))]
    p = [sum(x) / 4 for x in correct]
    flags = robust_flags(correct)
    raw = raw_flags(correct, [q["base_first_correct"] for q in bundle["anchors"]])
    return ctrl.update(flags, raw, step=step,
        extra={"anchor_p": p, "anchor_acc_held": sum(p[i] for i in bundle["held_idx"]) / len(bundle["held_idx"]),
               "anchor_acc_repair": sum(p[i] for i in bundle["repair_idx"]) / len(bundle["repair_idx"]),
               "capped": sum(r["capped"] for r in records) / len(records), "records": records})


def replay(rows, bundle, ctrl, c, ptr, arm="grpo_pi_nafc"):
    if arm == "grpo_pi_fixed_replay":
        pool = bundle["repair_idx"]
        n = min(math.ceil(c["fixed_replay_fraction"] * len(rows)), len(pool))
        return replace_groups(rows, n, pool, ptr, lambda old, i: bundle["anchors"][i], key=lambda x: x["id"])
    pool = ctrl.replay_pool(len(bundle["anchors"]))
    n = n_replay_groups(ctrl.lam, c["replay_frac_max"], len(rows), len(pool)) if ctrl.pulling else 0
    if n == 0:
        return rows, ptr, []
    return replace_groups(rows, n, pool, ptr, lambda old, i: bundle["anchors"][i], key=lambda x: x["id"])


def save_checkpoint(path, model, optimizer, scheduler, scaler, state):
    value = {"parameters": {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad},
             "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
             "scaler": scaler.state_dict(), "state": state}
    tmp = Path(str(path) + ".tmp")
    torch.save(value, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    params = dict(model.named_parameters())
    expected = {n for n, p in params.items() if p.requires_grad}
    if set(saved["parameters"]) != expected:
        raise ValueError("Checkpoint trainable parameters differ.")
    with torch.no_grad():
        for n, p in saved["parameters"].items():
            params[n].copy_(p.to(params[n].device))
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    scaler.load_state_dict(saved["scaler"])
    return saved["state"]


def train_arm(c, data, bundle, arm, loader=load_model, sampler_factory=Sampler):
    if arm not in ARMS:
        raise ValueError("Unknown arm")
    folder = OUT / arm
    folder.mkdir(exist_ok=True)
    if (folder / "complete.json").exists():
        print(f"[{arm}] Complete; use a new outputs folder to start a different run.", flush=True)
        return
    if not (folder / "resume.pt").exists() and any(folder.iterdir()):
        raise ValueError(f"{folder} contains a partial run without a checkpoint; do not silently restart it.")
    model, tok = loader(c, c.get("reference_checkpoint", OUT / "reference_model"), trainable=True)
    model.train()
    sampler = sampler_factory(model, tok, c)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=c["learning_rate"], weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda t: min(t / c["warmup_steps"], 1.0))
    scaler = torch.cuda.amp.GradScaler(init_scale=c.get("loss_scale", 65536),
                                      enabled=c["precision"] == "fp16" and torch.cuda.is_available())
    pi = LagrangianController("laconic", budgets=[bundle["budget"]], max_len=c["max_new_tokens"],
                              eta=c["pi_eta"], kp=c["pi_kp"], two_sided=True)
    pi.ceiling = lambda k: c["pi_lambda_max"]
    nafc = FlipController(mode="nafc_replay" if arm == "grpo_pi_nafc" else "monitor", eps=bundle["eps"],
                          eta=c["nafc_eta"], lam_max=c["nafc_lambda_max"],
                          repair_idx=bundle["repair_idx"], held_idx=bundle["held_idx"])
    state = {"step": 0, "replay_ptr": 0, "last_audit_step": None, "metrics": [], "audits": [], "validation": {}}
    if (folder / "resume.pt").exists():
        state = load_checkpoint(folder / "resume.pt", model, optimizer, scheduler, scaler)
        pi.load_state_dict(state["pi"]); nafc.load_state_dict(state["nafc"])
        print(f"[{arm}] Resuming after batch {state['step']}.", flush=True)
    paired = schedule(data["task_train"], c["steps"], c["prompts_per_step"], c["seed"])
    def checkpoint():
        state.update(pi=pi.state_dict(), nafc=nafc.state_dict())
        save_checkpoint(folder / "resume.pt", model, optimizer, scheduler, scaler, state)
        write(folder / "train_metrics.json", state["metrics"])
        write(folder / "audit_metrics.json", state["audits"])
    def validation(step):
        for domain in ("numina",):
            key = f"{domain}_{step}"
            if key not in state["validation"]:
                records = evaluate(sampler, data[f"{domain}_validation"], 4, 600_000_000 + step * 1_000_000,
                                   folder / f"validation_{key}.json")
                state["validation"][key] = {"accuracy": sum(r["correct"] for r in records) / len(records),
                    "mean_tokens": sum(len(r["tokens"]) for r in records) / len(records)}
        checkpoint()
    def midpoint(step):
        # Save exact weights/state before a resumable audit so training cannot skip an interrupted milestone.
        checkpoint()
        model.save_pretrained(folder / 'adapters' / f'step_{step}')
        test_sampler = sampler_factory(model, tok, dict(c, do_sample=c.get('test_do_sample', True)))
        for domain in evaluation_domains(data):
            evaluate(test_sampler, data[f'{domain}_test'], c.get('test_draws', 4), 430_000_000,
                     test_path(folder, step, c['steps'], domain))
    midpoints = set(c.get('test_steps', ())) - {c['steps']}
    if not (folder / "resume.pt").exists():
        checkpoint()
    if state["step"] in c["validation_steps"]:
        validation(state["step"])
    if state['step'] in midpoints:
        midpoint(state['step'])
    for step in range(state["step"], c["steps"]):
        started = time.monotonic()
        rows, state["replay_ptr"], used = replay(paired[step], bundle, nafc, c, state["replay_ptr"], arm)
        records = sampler(rows, c["group_size"], 700_000_000 + step * 1_000_000 + c["seed"])
        expanded = [q for q in rows for _ in range(c["group_size"])]
        lengths = [len(r["tokens"]) for r in records]
        if len(records) != len(expanded) or any(r["id"] != q["id"] for r, q in zip(records, expanded)):
            raise ValueError("Incomplete or reordered task sampling")
        lam = pi.lambdas[0]
        adv = decoupled_advantages([q["id"] for q in expanded], [float(r["correct"]) for r in records],
                                   [(L - bundle["budget"]) / bundle["budget"] for L in lengths], [lam] * len(records))
        pi.dual_update(lengths, [0] * len(records))
        if step % c["audit_every"] == 0 and state["last_audit_step"] != step:
            full = audit(sampler, bundle, nafc, c, step)
            write(folder / f"audit_step_{step}.json", full)
            state["audits"].append({k: v for k, v in full.items() if k != "records"})
            nafc.history[-1].pop("records", None)
            state["last_audit_step"] = step
            print(f"[{arm}] audit {step}: held F={full['F_used']:.3%}, NAFC lambda={nafc.lam:.3f}", flush=True)
        optimizer.zero_grad(set_to_none=True)
        loss_total = 0.0
        active = False
        lr_used = optimizer.param_groups[0]["lr"]
        for q, record, advantage in zip(expanded, records, adv):
            if abs(advantage) < 1e-12:
                continue
            active = True
            logps = token_logps(model, prompt_ids(tok, q["question"]), record["tokens"])
            # One on-policy update: the detached current scores are the exact pre-update old-policy scores.
            loss = clipped_loss(logps, logps.detach(), advantage, c["max_new_tokens"]) / len(records)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite policy loss; last complete checkpoint is preserved.")
            scaler.scale(loss).backward()
            loss_total += float(loss.detach())
        if not active:
            for p in params:
                p.grad = torch.zeros_like(p)
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        if not scaler.is_enabled() and not torch.isfinite(norm):
            raise FloatingPointError("Non-finite unscaled gradients")
        previous = scaler.get_scale()
        scaler.step(optimizer); scaler.update()
        updated = scaler.get_scale() >= previous
        if updated:
            scheduler.step()
        else:
            print(f"[{arm}] FP16 overflow: optimizer update skipped.", flush=True)
        state["step"] = step + 1
        state["metrics"].append({"step": step + 1, "loss": loss_total, "optimizer_updated": updated,
              "has_advantage_signal": active, "learning_rate": lr_used, "task_prompts": len(rows) - len(used),
              "anchor_prompts": len(used), "pi_lambda_used": lam, "pi_lambda_next": pi.lambdas[0],
              "nafc_lambda": nafc.lam, "accuracy": sum(r["correct"] for r in records) / len(records),
              "mean_tokens": sum(lengths) / len(lengths), "sampled_tokens": sum(lengths),
              "capped": sum(r["capped"] for r in records) / len(records), "seconds": time.monotonic() - started})
        print(f"[{arm}] {step + 1}/{c['steps']} acc={state['metrics'][-1]['accuracy']:.3f} "
              f"L={state['metrics'][-1]['mean_tokens']:.0f} replay={len(used)}", flush=True)
        if step + 1 in midpoints:
            midpoint(step + 1)
        elif (step + 1) % c.get("checkpoint_every", 25) == 0 or step + 1 == c["steps"]:
            checkpoint()
        if step + 1 in c["validation_steps"]:
            model.save_pretrained(folder / "adapters" / f"step_{step + 1}")
            validation(step + 1)
    model.save_pretrained(folder / "adapter_final")
    tok.save_pretrained(folder / "adapter_final")
    test_sampler = sampler_factory(model, tok, dict(c, do_sample=c.get("test_do_sample", True)))
    for domain in evaluation_domains(data):
        evaluate(test_sampler, data[f"{domain}_test"], c.get("test_draws", 4), 430_000_000,
                 test_path(folder, c['steps'], c['steps'], domain))
    if c.get("long_evaluation", True):
        long_sampler = sampler_factory(model, tok, dict(c, max_new_tokens=c["long_eval_tokens"]))
        for domain in evaluation_domains(data):
            evaluate(long_sampler, data[f"{domain}_test"], 4, 440_000_000,
                     folder / f"long_{domain}_test.json")
    write(folder / "complete.json", {"arm": arm, "step": state["step"], "validation": state["validation"],
          "test_steps": c.get('test_steps', [c['steps']]),
          "optimizer_updates": sum(x["optimizer_updated"] for x in state["metrics"]),
          "task_prompts": sum(x["task_prompts"] for x in state["metrics"]),
          "anchor_prompts": sum(x["anchor_prompts"] for x in state["metrics"]),
          "sampled_tokens": sum(x["sampled_tokens"] for x in state["metrics"]),
          "pi": pi.state_dict(), "nafc": nafc.state_dict()})


### `report.py`

In [ ]:
%%writefile /content/nafc_a100/report.py
"""Paired per-seed results and uncertainty across all prespecified seeds."""
import json
import numpy as np
from io_utils import ROOT, OUT, read, write
from data import evaluation_domains


def question_rates(records, questions, n):
    if len(records) != len(questions)*n:
        raise ValueError("Incomplete evaluation; do not compare partial arms.")
    fields = {k: [] for k in ("accuracy", "tokens", "capped", "no_box")}
    for i, q in enumerate(questions):
        group = records[i*n:(i+1)*n]
        if [(r["id"], r["sample_index"]) for r in group] != [(q["id"], j) for j in range(n)]:
            raise ValueError("Evaluation question/sample order mismatch")
        fields["accuracy"].append(sum(r["correct"] for r in group)/n)
        fields["tokens"].append(sum(len(r["tokens"]) for r in group)/n)
        for k in ("capped", "no_box"):
            fields[k].append(sum(r[k] for r in group)/n)
    return {k: np.asarray(v) for k, v in fields.items()}


def retention(initial, current, cohort=None):
    cohort = initial >= .75 if cohort is None else cohort
    n = int(cohort.sum()); bad = int((cohort & (current <= .25)).sum())
    return {"initially_stable": n, "correct_to_low_success": bad, "rate": bad/n if n else None}


def paired_interval(values, seed=0):
    if not len(values):
        return None
    rng = np.random.default_rng(seed)
    # Chunk bootstrap draws to avoid allocating a samples-by-full-benchmark matrix.
    means = [values[rng.integers(0, len(values), (100, len(values)))].mean(axis=1) for _ in range(40)]
    return np.percentile(np.concatenate(means), [2.5, 97.5]).tolist()


def make_report(shared):
    from engine import ARMS
    data = read(shared / "splits.json")
    bundle = read(shared / "prepared.json")
    payload = {"seed": read(OUT / "identity.json")["settings"]["seed"],
               "data_provenance": data["provenance"], "results": {}, "differences": {}, "warnings": []}
    for arm in ARMS:
        if not (OUT / arm / "complete.json").exists():
            raise ValueError(f"{arm} incomplete; all three arms are required.")
        payload["results"][arm] = {"run": read(OUT / arm / "complete.json")}
    for limit in ("primary", "long"):
        prefix = "" if limit == "primary" else "long_"
        for domain in evaluation_domains(data):
            questions = data[f"{domain}_test"]
            initial = question_rates(read(shared / f"reference_{prefix}{domain}_test.json")["records"], questions, 8)
            noise = question_rates(read(shared / f"reference_{prefix}{domain}_noise.json")["records"], questions, 4)
            stable = initial["accuracy"] >= .75
            raw = {}
            endpoint = f"{limit}_{domain}"
            payload[endpoint] = {"reference": {k: float(v.mean()) for k, v in initial.items()},
                                 "unchanged_noise": retention(initial["accuracy"], noise["accuracy"])}
            for arm in ARMS:
                name = "final" if limit == "primary" else "long"
                current = question_rates(read(OUT / arm / f"{name}_{domain}_test.json")["records"], questions, 4)
                raw[arm] = current
                gained_cohort = initial["accuracy"] <= .25
                gained_n = int(gained_cohort.sum())
                payload["results"][arm][endpoint] = {
                    **{k: float(v.mean()) for k, v in current.items()},
                    "retention": retention(initial["accuracy"], current["accuracy"]),
                    "newly_correct": int((gained_cohort & (current["accuracy"] >= .75)).sum()),
                    "initially_low_success": gained_n,
                    "regressions_by_topic": {topic: retention(initial["accuracy"], current["accuracy"],
                        stable & np.asarray([q["topic"] == topic for q in questions]))
                        for topic in sorted({q["topic"] for q in questions})}}
            for control in ARMS[:-1]:
                a, b = raw["grpo_pi_nafc"]["accuracy"], raw[control]["accuracy"]
                flip = (a[stable] <= .25).astype(float) - (b[stable] <= .25).astype(float)
                payload["differences"][f"{endpoint}_vs_{control}"] = {
                    "accuracy": float((a-b).mean()), "accuracy_question_ci95": paired_interval(a-b),
                    "flip_rate": float(flip.mean()) if len(flip) else None,
                    "flip_question_ci95": paired_interval(flip)}
            if stable.sum() < 128:
                payload["warnings"].append(f"{endpoint}: only {stable.sum()} initially stable test questions; coarse retention estimate.")
    if payload["results"]["grpo_pi_nafc"]["run"]["anchor_prompts"] == 0:
        payload["warnings"].append("NAFC never replayed; this seed cannot demonstrate active protection.")
    for arm in ARMS:
        info = payload["results"][arm]["run"]
        if info["optimizer_updates"] != info["step"]:
            payload["warnings"].append(f"{arm}: skipped optimizer updates; examine logs.")
    payload["warnings"] += ["Omni scores cover the numeric-answer subset; not the full official leaderboard protocol.",
        "Numina is a research holdout. Olympic source labels are not a certificate that every question is IMO-level.",
        "Deduplication does not certify absence of paraphrases or model pretraining contamination.",
        "Fixed replay uses one prespecified fraction, not a best-of-grid baseline."]
    write(OUT / "comparison.json", payload)
    lines = ["# Phi Olympic results", "", f"Training seed: {payload['seed']}. Final checkpoint only.", "",
             "| Endpoint | Arm | Accuracy | Correct → low success | Tokens | Capped |",
             "|---|---|---:|---:|---:|---:|"]
    for endpoint in ("primary_numina", "primary_omni", "long_numina", "long_omni"):
        for arm in ARMS:
            r = payload["results"][arm][endpoint]; f = r["retention"]
            lines.append(f"| {endpoint} | {arm} | {r['accuracy']:.2%} | {f['correct_to_low_success']}/{f['initially_stable']} | {r['tokens']:.1f} | {r['capped']:.2%} |")
    lines += ["", "Capped and unboxed responses remain in every primary denominator.",
              f"Noise allowance epsilon={bundle['eps']:.3%}; PI budget B={bundle['budget']}.",
              "Question intervals are conditional on this training seed; use SUMMARY.md for seed variation.",
              "", *[f"- {w}" for w in payload["warnings"]]]
    (OUT / "REPORT.md").write_text("\n".join(lines)+"\n", encoding="utf-8")
    plots(payload)
    print("\n".join(lines), flush=True)
    return payload


def plots(payload):
    from engine import ARMS
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, domain in zip(axes, ("numina", "omni")):
        endpoint = f"primary_{domain}"
        ax.bar(["Reference", *ARMS], [100*payload[endpoint]["reference"]["accuracy"],
               *[100*payload["results"][a][endpoint]["accuracy"] for a in ARMS]])
        ax.set(title=f"{domain}: numeric-answer subset", ylabel="Accuracy (%)", ylim=(0, 100))
        ax.tick_params(axis="x", rotation=20)
    fig.tight_layout(); fig.savefig(OUT / "accuracy.png", dpi=160); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, domain in zip(axes, ("numina", "omni")):
        for arm in ARMS:
            r = payload["results"][arm][f"primary_{domain}"]["retention"]
            ax.bar(arm, r["correct_to_low_success"])
        ax.set(title=f"{domain}: initially stable n={r['initially_stable']}", ylabel="Regressed questions")
        ax.tick_params(axis="x", rotation=20)
    fig.tight_layout(); fig.savefig(OUT / "regressions.png", dpi=160); plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for arm in ARMS:
        m = read(OUT / arm / "train_metrics.json"); a = read(OUT / arm / "audit_metrics.json")
        axes[0].plot([r["step"] for r in m], [r["pi_lambda_used"] for r in m], label=arm)
        axes[1].plot([r["step"] for r in a], [r["F_used"] for r in a], label=arm)
        axes[2].plot([r["step"] for r in a], [r["lam"] for r in a], label=arm)
    for ax, title in zip(axes, ("PI multiplier", "Held-anchor flagged rate", "NAFC multiplier")):
        ax.set(title=title, xlabel="Training batch"); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "controllers.png", dpi=160); plt.close(fig)


def aggregate(c):
    root = ROOT / "outputs_phi_olympic"
    summaries = []
    for seed in c["seeds"]:
        path = root / f"seed_{seed}" / "comparison.json"
        if not path.exists():
            raise ValueError(f"Seed {seed} is incomplete. All prespecified seeds are required.")
        summaries.append(read(path))
    fingerprints = [read(root / f"seed_{s}" / "identity.json") for s in c["seeds"]]
    settings = [{k: v for k, v in x["settings"].items() if k not in ("seed", "reference_checkpoint")} for x in fingerprints]
    if any(x != settings[0] for x in settings) or any(
            (x["splits_sha256"], x["reference"], x["code_sha256"]) !=
            (fingerprints[0]["splits_sha256"], fingerprints[0]["reference"], fingerprints[0]["code_sha256"])
            for x in fingerprints):
        raise ValueError("Cannot aggregate different protocols, splits, references or code.")
    results = {}
    for endpoint in summaries[0]["differences"]:
        results[endpoint] = {}
        for metric in ("accuracy", "flip_rate"):
            values = [s["differences"][endpoint][metric] for s in summaries]
            if any(v is None for v in values):
                results[endpoint][metric] = {"mean": None, "ci95": None, "seeds": values}
            else:
                v = np.asarray(values)
                results[endpoint][metric] = {"mean": float(v.mean()), "ci95": paired_interval(v), "seeds": values}
    write(root / "summary.json", {"seeds": c["seeds"], "differences_nafc_minus_baseline": results,
        "scope": "Seed-bootstrap intervals conditional on one frozen data split. Five seeds do not establish universal effectiveness."})
    lines = ["# Phi Olympic multi-seed summary", "", f"All prespecified seeds: {c['seeds']}.", "",
             "| Comparison | Accuracy delta (pp) | 95% seed CI | Regression-rate delta (pp) | 95% seed CI |",
             "|---|---:|---|---:|---|"]
    def estimate(r):
        return "undefined" if r["mean"] is None else f"{100*r['mean']:+.2f}"
    def interval(r):
        return "undefined" if r["ci95"] is None else f"[{100*r['ci95'][0]:+.2f}, {100*r['ci95'][1]:+.2f}]"
    for key, r in results.items():
        lines.append(f"| {key} | {estimate(r['accuracy'])} | {interval(r['accuracy'])} | {estimate(r['flip_rate'])} | {interval(r['flip_rate'])} |")
    lines += ["", "Negative regression delta favours NAFC. Positive accuracy delta favours NAFC.",
              "Inspect unchanged-reference noise, active replay, task exposure, token cost and long-cap sensitivity before interpreting protection.",
              "No seed removal or best-checkpoint selection is used. These intervals are descriptive with only five independent seeds."]
    (root / "SUMMARY.md").write_text("\n".join(lines)+"\n", encoding="utf-8")
    print("\n".join(lines), flush=True)


### `colab_pilot.py`

In [ ]:
%%writefile /content/nafc_a100/colab_pilot.py
"""Paired multi-seed Olympic evaluation, resumable across Colab sessions."""
import argparse
import importlib.metadata
import os
import sys
from pathlib import Path
os.environ.setdefault("HF_HOME", str(Path(__file__).resolve().parent / "hf_cache"))
import torch
import engine
import io_utils
import model as modeling
from core import robust_flags, split_anchors
from scoring import math_gold
from data import DATASETS, rows, normalized, take
from io_utils import ROOT, read, write, freeze, digest, file_hash, checkpoint_identity
from model import Sampler, load_model, base_path, prompt_ids, token_logps, evaluate, clear_memory

ARMS = engine.PAIRED_ARMS
PROTOCOL = 'olympic-a100-manual-v6'
MODELS = {
    'microsoft/Phi-4-mini-instruct': 'cfbefacb99257ffa30c83adab238a50856ac3083',
    'google/gemma-3-1b-it': 'dcc83ea841ab6100d6b47a070329e1ba4cf78752'}
OUT = ROOT / "outputs_colab"


def configure(output):
    global OUT
    OUT = Path(output).resolve()
    OUT.mkdir(parents=True, exist_ok=True)
    engine.OUT = OUT
    return OUT


def validate(c):
    if c['base_model'] not in MODELS or c['base_revision'] != MODELS[c['base_model']] or c['precision'] != 'fp16':
        raise ValueError('Use the pinned Phi/Gemma model revision in fp16.')
    for k in ("steps", "prompts_per_step", "group_size", "train_count", "validation_count", "test_count",
              "anchor_pool", "anchor_count", "anchor_minimum", "generation_batch", "checkpoint_every"):
        if not isinstance(c[k], int) or c[k] <= 0:
            raise ValueError(f"Invalid positive count: {k}")
    if c["initial_draws"] != 8 or c["eval_draws"] != 4 or c["group_size"] < 2:
        raise ValueError("Preserve the eight-draw selection and four-draw NAFC audit.")
    if c["anchor_minimum"] < 4 or c["anchor_minimum"] % 2 or c["anchor_count"] % 2:
        raise ValueError("Repair and controller halves must each have at least two anchors.")
    if not c["anchor_minimum"] <= c["anchor_count"] <= c["anchor_pool"]:
        raise ValueError("Invalid declared anchor size range")
    if c["test_draws"] != 4 or not c["test_do_sample"] or not c["long_evaluation"]:
        raise ValueError("Preserve four-draw stochastic evaluation and long-cap sensitivity.")
    if c['long_eval_tokens'] <= c['max_new_tokens']:
        raise ValueError('The secondary evaluation must use a larger token cap.')
    if c['max_prompt_tokens'] + c['long_eval_tokens'] > c.get('context_length', 131072):
        raise ValueError('Prompt plus long response exceeds the model context; no silent truncation.')
    if len(c['seeds']) < 3 or len(set(c['seeds'])) != len(c['seeds']) or c['seed'] not in c['seeds']:
        raise ValueError('Use all prespecified independent training seeds.')
    if not 0 < c['replay_frac_max'] < 1:
        raise ValueError('Invalid maximum NAFC group replacement fraction.')
    milestones = c['test_steps']
    if not milestones or milestones != sorted(set(milestones)) or milestones[-1] != c['steps'] or any(
            not isinstance(s, int) or s <= 0 or s > c['steps'] for s in milestones):
        raise ValueError('Test milestones must be ordered, unique, within the training budget and include its final step.')
    if c['anchor_minimum'] != c['anchor_count']:
        raise ValueError('This evaluation requires the full declared anchor set; no smaller denominator.')
    if c['train_count'] < c['prompts_per_step'] or c['train_count'] % c['prompts_per_step']:
        raise ValueError('Training set must form complete prompt groups; epochs repeat with a seeded shuffle.')
    if file_hash(ROOT/"core.py") != read(ROOT/"SOURCE.json")["core_sha256"]:
        raise ValueError("Upstream PI/NAFC core changed")


def hardware(c):
    modeling.check_gpu(c)
    memory = torch.cuda.get_device_properties(0).total_memory/(1024**3)
    if c.get('target_gpu') == 'A100' and ('A100' not in torch.cuda.get_device_name(0) or memory < 35):
        raise RuntimeError('Select an A100 runtime for these notebooks; no automatic hardware downgrade.')
    if memory < 14:
        raise RuntimeError("This fp16 profile requires a GPU with at least 14 GiB usable VRAM, such as Colab T4.")
    print(f"[GPU] {torch.cuda.get_device_name(0)}: {memory:.1f} GiB; fp16; chunked output-head loss", flush=True)


def prepare_data(c):
    frozen = ROOT/c.get('data_file', 'colab_splits.json')
    if frozen.exists():
        saved = read(frozen)
        settings = data_settings(c)
        if saved['settings'] != settings:
            raise ValueError('Frozen notebook data does not match its declared split configuration.')
        data = saved['splits']
        validate_data(data, c)
        print('[data] Using embedded frozen questions; no full-corpus download needed.', flush=True)
        return data
    return build_data(c)


def build_data(c):
    from datasets import load_dataset
    if c.get('test_domain') == 'math':
        import hashlib
        name = 'DigitalLearningGmbH/MATH-lighteval'
        original = {s: load_dataset(name, revision=c['math_revision'], split=s) for s in ('train', 'test')}
        pools, invalid_labels = {}, {}
        for split, dataset in original.items():
            unique = {}; invalid_labels[split] = []
            for i, item in enumerate(dataset):
                key = normalized(item['problem'])
                answer = math_gold(item['solution'])
                if not key:
                    raise ValueError('MATH contains an empty question.')
                if not answer:
                    invalid_labels[split].append(i)
                    continue
                unique.setdefault(key, {'id': hashlib.sha256(f'math/{split}/{key}'.encode()).hexdigest(),
                    'domain': 'math', 'question': item['problem'], 'answer': answer, 'topic': item['type'],
                    'source': name, 'source_index': i, 'difficulty': item.get('level')})
            pools[split] = list(unique.values())
        blocked = {normalized(q['question']) for q in pools['test']}
        pool = [q for q in pools['train'] if normalized(q['question']) not in blocked]
        a, b = c['train_count'], c['train_count']+c['validation_count']
        selected = take(pool, b+c['anchor_pool'], c['split_seed'])
        result = {'task_train': selected[:a], 'math_validation': selected[a:b], 'anchor_candidates': selected[b:],
            'math_test': take(pools['test'], c['test_count'], c['split_seed']+1),
            'provenance': {'task_domain': 'math', 'evaluation_domains': ['math'],
                'datasets': {'math': name}, 'revisions': {'math': c['math_revision']},
                'source_counts': {k: len(v) for k,v in original.items()},
                'excluded_missing_or_empty_gold': invalid_labels,
                'scope': 'Fixed random MATH test subset; all seven topics and answer types, no integer/difficulty/success filter.',
                'deduplication': 'Normalized exact train/test overlap removed before selection.'}}
        validate_data(result, c)
        return result
    external = load_dataset(DATASETS['omni'], revision=c['omni_revision'], split='test')
    blocked = {normalized(q['problem']) for q in external}
    eligible_test = rows(external, 'omni', 'test')
    raw = load_dataset(DATASETS['numina'], revision=c['numina_revision'], split='train')
    pool = [q for q in rows(raw, 'numina', 'research_pool', c['numina_sources'])
            if normalized(q['question']) not in blocked]
    picked = take(pool, c['train_count']+c['validation_count']+c['anchor_pool'], c['split_seed'])
    a = c['train_count']; b = a+c['validation_count']
    data = {'task_train': picked[:a], 'numina_validation': picked[a:b], 'anchor_candidates': picked[b:],
        'omni_test': take(eligible_test, c['test_count'], c['split_seed']+1),
        'provenance': {'task_domain': 'numina', 'evaluation_domains': ['omni'],
            'datasets': DATASETS, 'numina_sources': c['numina_sources'],
            'revisions': {k: c[f'{k}_revision'] for k in DATASETS},
            'eligible_numina': len(pool), 'eligible_omni': len(eligible_test), 'omni_total': len(external),
            'scope': 'Fixed random numeric-answer subset of Omni-MATH; not an official full-benchmark score.',
            'deduplication': 'Normalized exact overlap with ALL Omni questions removed; paraphrases not certified.'}}
    validate_data(data, c)
    return data


def data_settings(c):
    keys = ['split_seed', 'train_count', 'validation_count', 'anchor_pool', 'test_count']
    keys += ['math_revision'] if c.get('test_domain') == 'math' else ['numina_revision', 'omni_revision', 'numina_sources']
    return {k: c[k] for k in keys}


def validate_data(data, c):
    domain = c.get('test_domain', 'omni')
    validation = 'math_validation' if domain == 'math' else 'numina_validation'
    for name, count in (('task_train', c['train_count']), (validation, c['validation_count']),
                        ('anchor_candidates', c['anchor_pool']), (f'{domain}_test', c['test_count'])):
        if len(data[name]) != count or any(not q['answer'] for q in data[name]):
            raise ValueError(f'Invalid frozen question count or answer in {name}')
    all_rows = [q for name, rr in data.items() if name != 'provenance' for q in rr]
    keys = [normalized(q['question']) for q in all_rows]
    ids = [q['id'] for q in all_rows]
    if len(keys) != len(set(keys)) or len(ids) != len(set(ids)):
        raise ValueError("Question leakage between pilot partitions")


def identity(c, data, reference):
    return {'protocol': PROTOCOL, 'config': c, 'splits': digest(data),
        'reference': reference, 'code': {name: file_hash(ROOT/name) for name in
        ('core.py', 'data.py', 'engine.py', 'model.py', 'io_utils.py', 'scoring.py', 'colab_pilot.py', 'report.py', 'colab_report.py', 'manual_artifacts.py')},
        'versions': {k: importlib.metadata.version(k) for k in ('torch', 'transformers', 'peft', 'math-verify', 'accelerate', 'numpy', 'matplotlib')},
        'gpu': torch.cuda.get_device_name(0)}


def dry_backward(model, tok, c):
    # Validate the worst configured response length before expensive calibration, with zero optimizer updates.
    model.train(); model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    ids = prompt_ids(tok, 'Calculate 2 + 2.')
    token = tok.encode('1', add_special_tokens=False)[-1]
    logps = token_logps(model, ids, [token]*c['max_new_tokens'])
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=c['learning_rate'])
    scaler = torch.cuda.amp.GradScaler(init_scale=c['loss_scale'])
    scaler.scale(-logps.mean()).backward(); scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    if not torch.isfinite(norm):
        raise FloatingPointError("Non-finite preflight gradients; no policy was updated.")
    print(f"[preflight] {c['max_new_tokens']}-token backward peak: {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB", flush=True)
    model.zero_grad(set_to_none=True)
    del optimizer, scaler, logps


def select_anchors(c, data, sampler, shared):
    path = shared/'anchor_selection.json'
    state = read(path) if path.exists() else {'tested': 0, 'records': []}
    pool = data['anchor_candidates']; n = 8
    if [(r['id'], r['sample_index']) for r in state['records']] != [
        (q['id'], j) for q in pool[:state['tested']] for j in range(n)]:
        raise ValueError("Incomplete or reordered anchor-selection progress")
    def stable():
        return [dict(q, base_first_correct=state['records'][i*n]['correct'])
            for i, q in enumerate(pool[:state['tested']])
            if sum(r['correct'] for r in state['records'][i*n:(i+1)*n]) >= 6]
    anchors = stable()
    while len(anchors) < c['anchor_count'] and state['tested'] < len(pool):
        start = state['tested']; batch = pool[start:start+c['anchor_selection_batch']]
        rr = sampler(batch, n, 300_000_000+start*n*1009)
        if [(r['id'], r['sample_index']) for r in rr] != [(q['id'], j) for q in batch for j in range(n)]:
            raise ValueError("Incomplete anchor generation")
        state['records'].extend(rr); state['tested'] += len(batch)
        anchors = stable()
        write(path, state)
        print(f"[anchors] stable={len(anchors)}, target={c['anchor_count']}, tested={state['tested']}/{len(pool)}", flush=True)
    return anchors[:c['anchor_count']] if len(anchors) >= c['anchor_count'] else []


def prepare(c):
    hardware(c)
    shared = OUT/'shared'; shared.mkdir(exist_ok=True)
    freeze(shared/'config.json', c)
    data = read(shared/'splits.json') if (shared/'splits.json').exists() else prepare_data(c)
    freeze(shared/'splits.json', data)
    path = base_path(c)
    freeze(shared/'identity.json', identity(c, data, checkpoint_identity(path)))
    if (shared/'prepared.json').exists():
        print('[prepare] Reusing complete preparation.', flush=True)
        return read(shared/'prepared.json')
    model, tok = load_model(c, path, trainable=True)
    for name, rr in data.items():
        if name != 'provenance' and any(len(prompt_ids(tok, q['question'])) > c['max_prompt_tokens'] for q in rr):
            raise ValueError('A prompt exceeds the cap; no silent truncation.')
    dry_backward(model, tok, c)
    sampler = Sampler(model, tok, c)
    validation = 'math_validation' if c.get('test_domain') == 'math' else 'numina_validation'
    val = evaluate(sampler, data[validation], 4, 410_000_000, shared/'reference_validation.json')
    mean = sum(len(r['tokens']) for r in val)/len(val)
    anchors = select_anchors(c, data, sampler, shared)
    if not anchors:
        diagnostic = {'status': 'insufficient_stable_anchors', 'minimum': c['anchor_minimum'],
            'note': 'Frozen Olympic pool did not supply enough stable anchors. No arm was trained; no efficacy comparison is valid.'}
        write(shared/'diagnostic.json', diagnostic)
        print(diagnostic, flush=True)
        del sampler, model, tok; clear_memory()
        return diagnostic
    repair, held = split_anchors(len(anchors), .5, 0)
    noise = []
    for repeat in range(3):
        rr = evaluate(sampler, [anchors[i] for i in held], 4, 400_000_000+repeat*1_000_000,
                      shared/f'anchor_noise_{repeat}.json')
        noise.append(sum(robust_flags([[r['correct'] for r in rr[i*4:(i+1)*4]] for i in range(len(held))]))/len(held))
    domain = c.get('test_domain', 'omni')
    for prefix, cap in (('', c['max_new_tokens']), ('long_', c['long_eval_tokens'])):
        test_sampler = Sampler(model, tok, dict(c, max_new_tokens=cap, do_sample=True))
        initial = evaluate(test_sampler, data[f'{domain}_test'], 8, 420_000_000 if not prefix else 450_000_000,
                          shared/f'reference_{prefix}{domain}_test.json')
        evaluate(test_sampler, data[f'{domain}_test'], 4, 430_000_000 if not prefix else 440_000_000,
                 shared/f'reference_{prefix}{domain}_noise.json')
        stable = sum(sum(r['correct'] for r in initial[i*8:(i+1)*8]) >= 6 for i in range(len(data[f'{domain}_test'])))
        print(f'[reference {cap}] initially stable={stable}/{len(data[f"{domain}_test"])}', flush=True)
        if stable < 128:
            print('[warning] Fewer than 128 stable test questions: retention precision remains limited. No test rows are removed.', flush=True)
    bundle = {'status': 'prepared', 'anchors': anchors, 'repair_idx': repair, 'held_idx': held,
        'noise_repeats': noise, 'noise_robust': sum(noise)/3, 'eps': sum(noise)/3+c['noise_margin'],
        'budget': max(1, round(mean*c['compression_ratio'])), 'calibration_mean_tokens': mean,
        'anchor_target': c['anchor_count'], 'anchor_actual': len(anchors),
        'identity_sha256': digest(read(shared/'identity.json'))}
    write(shared/'prepared.json', bundle)
    print(f"[prepare] actual anchors={len(anchors)}, B={bundle['budget']}, noise={bundle['noise_robust']:.2%}, epsilon={bundle['eps']:.2%}", flush=True)
    del test_sampler, sampler, model, tok; clear_memory()
    return bundle


def train(c, arm, seed):
    hardware(c)
    shared = OUT/'shared'
    data, bundle = read(shared/'splits.json'), read(shared/'prepared.json')
    path = base_path(c)
    if digest(identity(c, data, checkpoint_identity(path))) != bundle['identity_sha256']:
        raise ValueError('Prepared model, code, environment or configuration changed; refusing mismatched resume.')
    seeded = dict(c, seed=seed)
    folder = OUT/f'seed_{seed}'; folder.mkdir(exist_ok=True)
    freeze(folder/'identity.json', identity(seeded, data, checkpoint_identity(path)))
    engine.OUT = folder
    engine.train_arm(dict(seeded, reference_checkpoint=path), data, bundle, arm)


def report():
    from colab_report import report as summarize
    return summarize(OUT)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('command', choices=['prepare', 'train', 'report', 'all'])
    parser.add_argument('--output', default=str(OUT))
    parser.add_argument('--arm', choices=ARMS)
    parser.add_argument('--seed', type=int)
    parser.add_argument('--config', default=str(ROOT/'config_colab.json'))
    args = parser.parse_args()
    c = read(args.config); validate(c); configure(args.output)
    if args.command == 'prepare': prepare(c)
    elif args.command == 'train':
        if not args.arm or args.seed not in c['seeds']:
            parser.error('train requires --arm and a prespecified --seed')
        train(c, args.arm, args.seed)
    elif args.command == 'report': report()
    else:
        import subprocess
        base = [sys.executable, str(ROOT/'colab_pilot.py'), '--config', args.config]
        subprocess.run(base+['prepare', '--output', str(OUT)], check=True)
        if not (OUT/'shared'/'prepared.json').exists(): return
        for seed in c['seeds']:
            for arm in ARMS:
                subprocess.run(base+['train', '--arm', arm, '--seed', str(seed), '--output', str(OUT)], check=True)
        report()


if __name__ == '__main__': main()


### `colab_report.py`

In [ ]:
%%writefile /content/nafc_a100/colab_report.py
"""All-seed paired evaluation; robust retention and token-cap sensitivity."""
import numpy as np
from io_utils import read, write
from report import question_rates, retention, paired_interval
from engine import PAIRED_ARMS as ARMS, test_path


def report(root):
    shared = root/'shared'
    c = read(shared/'config.json'); data = read(shared/'splits.json'); bundle = read(shared/'prepared.json')
    identity = read(shared/'identity.json')
    seeds = c['seeds']
    domain = c.get('test_domain', 'omni')
    runs, warnings = {}, []
    for seed in seeds:
        folder = root/f'seed_{seed}'
        expected = dict(identity, config=dict(c, seed=seed))
        if not (folder/'identity.json').exists() or read(folder/'identity.json') != expected:
            raise ValueError(f'Seed {seed}: mismatched or missing experiment identity.')
        runs[seed] = {}
        for arm in ARMS:
            path = folder/arm/'complete.json'
            if not path.exists() or read(path)['step'] != c['steps']:
                raise ValueError('All prespecified seeds and arms must finish the planned budget before comparison.')
            runs[seed][arm] = read(path)
            if runs[seed][arm].get('test_steps') != c['test_steps']:
                raise ValueError('Completed run does not match the prespecified test milestones.')
            if runs[seed][arm]['optimizer_updates'] != c['steps']:
                warnings.append(f'Seed {seed}, {arm}: skipped optimizer updates; examine actual update budgets.')
        if runs[seed]['grpo_pi_nafc']['anchor_prompts'] == 0:
            warnings.append(f'Seed {seed}: NAFC never replayed; active protection was not demonstrated.')
    endpoints, trajectory = {}, {}
    primary_initial = None
    for limit in ('primary', 'long'):
        prefix = '' if limit == 'primary' else 'long_'
        questions = data[f'{domain}_test']
        initial = question_rates(read(shared/f'reference_{prefix}{domain}_test.json')['records'], questions, 8)
        noise = question_rates(read(shared/f'reference_{prefix}{domain}_noise.json')['records'], questions, 4)
        stable = initial['accuracy'] >= .75
        if limit == 'primary': primary_initial = initial['accuracy']
        low = initial['accuracy'] <= .25
        noise_retention = retention(initial['accuracy'], noise['accuracy'])
        for step in c['test_steps'] if limit == 'primary' else [c['steps']]:
            endpoint = {'reference': {k: float(v.mean()) for k, v in initial.items()},
                        'unchanged_noise': noise_retention, 'initially_low_success': int(low.sum()),
                        'seeds': {}, 'comparisons': {}}
            raw = {}
            for seed in seeds:
                endpoint['seeds'][str(seed)] = {}
                for arm in ARMS:
                    folder = root/f'seed_{seed}'/arm
                    path = test_path(folder, step, c['steps'], domain) if limit == 'primary' else folder/f'long_{domain}_test.json'
                    if not path.exists():
                        raise ValueError(f'Missing prespecified test milestone: {path}')
                    rr = read(path)['records']
                    current = question_rates(rr, questions, 4)
                    raw[seed, arm] = current
                    f = retention(initial['accuracy'], current['accuracy'])
                    endpoint['seeds'][str(seed)][arm] = {
                        **{k: float(v.mean()) for k, v in current.items()}, 'retention': f,
                        'excess_flip_rate': None if f['rate'] is None else f['rate']-noise_retention['rate'],
                        'newly_reliable': int((low & (current['accuracy'] >= .75)).sum()),
                        'fix_rate': float((current['accuracy'][low] >= .75).mean()) if low.any() else None,
                        'regressed_with_capped_response': int((stable & (current['accuracy'] <= .25) & (current['capped'] > 0)).sum()),
                        'primary_cohort_retention': retention(primary_initial, current['accuracy']),
                        'regressions_by_topic': {topic: retention(initial['accuracy'], current['accuracy'],
                            stable & np.asarray([q['topic'] == topic for q in questions]))
                            for topic in sorted({q['topic'] for q in questions})}}
            for control in ARMS[:-1]:
                accuracy = np.stack([raw[s, 'grpo_pi_nafc']['accuracy']-raw[s, control]['accuracy'] for s in seeds])
                flips = np.stack([(raw[s, 'grpo_pi_nafc']['accuracy'][stable] <= .25).astype(float)-
                                  (raw[s, control]['accuracy'][stable] <= .25).astype(float) for s in seeds])
                def describe(values):
                    if values.shape[1] == 0:
                        return {'mean': None, 'seed_values': [None]*len(seeds), 'seed_sd': None, 'question_ci95': None}
                    per_seed = values.mean(axis=1)
                    return {'mean': float(per_seed.mean()), 'seed_values': per_seed.tolist(),
                            'seed_sd': float(per_seed.std(ddof=1)),
                            'question_ci95': paired_interval(values.mean(axis=0))}
                endpoint['comparisons'][control] = {'accuracy': describe(accuracy), 'flip_rate': describe(flips)}
            if stable.sum() < 128:
                warnings.append(f'{limit} step {step}: only {stable.sum()} initially stable questions; more steps cannot fix a small retention denominator.')
            if noise_retention['rate'] is not None and all(
                    endpoint['seeds'][str(s)]['grpo_pi']['retention']['rate'] <= noise_retention['rate'] for s in seeds):
                warnings.append(f'{limit} step {step}: control forgetting did not exceed unchanged-reference noise in any seed; active protection evidence is limited.')
            endpoint['step'] = step
            if limit == 'primary': trajectory[str(step)] = endpoint
            if step == c['steps']: endpoints[limit] = endpoint
    warnings += [f'{len(seeds)} training seeds provide limited seed-variation evidence; question intervals are conditional on these seeds and one frozen split.',
        data['provenance'].get('scope', 'Fixed benchmark subset; not a full official leaderboard score.'),
        'All capped and unboxed responses remain in primary denominators. Long-cap evaluation is prespecified, not a replacement chosen after results.',
        'Exact deduplication does not certify absence of paraphrases or model pretraining contamination.',
        'All prespecified milestones are reported. Question intervals are pointwise descriptive intervals, not corrected for multiple comparisons.',
        'This paired experiment tests the whole NAFC method; it does not isolate adaptive control from ordinary replay.']
    payload = {'protocol': identity['protocol'], 'model': c['base_model'], 'domain': domain, 'seeds': seeds, 'questions': len(data[f'{domain}_test']),
        'anchors': len(bundle['anchors']), 'allowance': bundle['eps'], 'pi_budget': bundle['budget'],
        'endpoints': endpoints, 'trajectory': trajectory, 'test_steps': c['test_steps'], 'runs': runs, 'warnings': warnings, 'data_provenance': data['provenance']}
    write(root/'comparison.json', payload)
    lines = [f'# {c["base_model"]}: {domain} evaluation', '', f'All prespecified seeds: {seeds}; primary test steps: {c["test_steps"]}. Long-cap sensitivity at the final step only.', '',
        '| Cap | Seed | Arm | Accuracy | Robust regressions | Newly reliable | Capped |',
        '|---|---:|---|---:|---:|---:|---:|']
    reported = [(f'primary step {step}', trajectory[str(step)]) for step in c['test_steps']] + [(f'long step {c["steps"]}', endpoints['long'])]
    for limit, endpoint in reported:
        ref = endpoint['reference']; noise = endpoint['unchanged_noise']
        lines.append(f"| {limit} | — | Reference (8 draws) | {ref['accuracy']:.2%} | unchanged noise {noise['correct_to_low_success']}/{noise['initially_stable']} | — | {ref['capped']:.2%} |")
        for seed in seeds:
            for arm, r in endpoint['seeds'][str(seed)].items():
                f = r['retention']
                lines.append(f"| {limit} | {seed} | {arm} | {r['accuracy']:.2%} | {f['correct_to_low_success']}/{f['initially_stable']} | {r['newly_reliable']}/{endpoint['initially_low_success']} | {r['capped']:.2%} |")
    lines += ['', '| Cap | Comparator | NAFC accuracy delta (pp) | NAFC regression delta (pp) |', '|---|---|---:|---:|']
    for limit, endpoint in reported:
        for control, d in endpoint['comparisons'].items():
            def estimate(v): return 'undefined' if v['mean'] is None else f"{100*v['mean']:+.2f} (seed SD {100*v['seed_sd']:.2f})"
            lines.append(f"| {limit} | {control} | {estimate(d['accuracy'])} | {estimate(d['flip_rate'])} |")
    lines += ['', 'Robust regression: initial success >=6/8, current success <=1/4. Newly reliable: initial <=2/8, current >=3/4.',
        'Negative regression delta and positive accuracy delta favour NAFC. Conditional paired question intervals and all seed values are in comparison.json.',
        'No seed deletion, best-checkpoint selection or pooling with the earlier small pilot.', '', *[f'- {w}' for w in warnings]]
    (root/'REPORT.md').write_text('\n'.join(lines)+'\n', encoding='utf-8')
    plot(root, payload)
    print('\n'.join(lines), flush=True)
    return payload


def plot(root, payload):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    results = payload['endpoints']['primary']['seeds']
    positions = np.arange(len(ARMS))
    labels = ['GRPO + PI', '+ NAFC']
    for ax, key, title in ((axes[0], 'accuracy', 'Sampled accuracy (%)'), (axes[1], 'retention', 'Robust regressions (%)')):
        values = np.asarray([[results[str(s)][a][key]['rate'] if key == 'retention' else results[str(s)][a][key]
                              for a in ARMS] for s in payload['seeds']], dtype=float)*100
        if np.isfinite(values).any():
            ax.bar(positions, values.mean(axis=0), yerr=values.std(axis=0, ddof=1), capsize=4)
            for i, row in enumerate(values): ax.scatter(positions+(i-1)*.07, row, label=f"seed {payload['seeds'][i]}", s=20)
        else:
            ax.text(.5, .5, 'No initially stable test questions', ha='center', transform=ax.transAxes)
        ax.set(xticks=positions, xticklabels=labels, ylabel=title); ax.tick_params(axis='x', rotation=15)
    axes[0].axhline(payload['endpoints']['primary']['reference']['accuracy']*100, color='black', linestyle='--', label='Reference')
    noise = payload['endpoints']['primary']['unchanged_noise']['rate']
    if noise is not None: axes[1].axhline(noise*100, color='black', linestyle='--', label='Unchanged noise')
    for seed in payload['seeds']:
        audits = read(root/f'seed_{seed}'/'grpo_pi_nafc'/'audit_metrics.json')
        axes[2].plot([a['step'] for a in audits], [a['lam'] for a in audits], label=f'seed {seed}')
    axes[2].set(title='NAFC multiplier', xlabel='Training batch', ylabel='Lambda')
    for ax in axes: ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(root/'overview.png', dpi=160); plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    steps = payload['test_steps']
    for arm, label in zip(ARMS, labels):
        for ax, metric, title in zip(axes, ('accuracy', 'retention', 'tokens'),
                                     ('Sampled accuracy (%)', 'Robust regressions (%)', 'Mean response tokens')):
            values = np.asarray([[payload['trajectory'][str(step)]['seeds'][str(seed)][arm][metric]['rate']
                                  if metric == 'retention' else payload['trajectory'][str(step)]['seeds'][str(seed)][arm][metric]
                                  for step in steps] for seed in payload['seeds']], dtype=float)
            if metric != 'tokens': values *= 100
            if np.isfinite(values).any():
                mean, sd = values.mean(axis=0), values.std(axis=0, ddof=1)
                line, = ax.plot(steps, mean, marker='o', label=label)
                ax.fill_between(steps, mean-sd, mean+sd, color=line.get_color(), alpha=.15)
            ax.set(xlabel='Training batch', ylabel=title, xticks=steps)
    axes[0].axhline(payload['endpoints']['primary']['reference']['accuracy']*100, color='black', linestyle='--', label='Reference')
    if noise is not None: axes[1].axhline(noise*100, color='black', linestyle='--', label='Unchanged noise')
    for ax in axes:
        if ax.get_legend_handles_labels()[0]: ax.legend(fontsize=8)
    fig.suptitle('All prespecified checkpoints; mean and training-seed SD')
    fig.tight_layout(); fig.savefig(root/'trajectory.png', dpi=160); plt.close(fig)


### `colab_checks.py`

In [ ]:
%%writefile /content/nafc_a100/colab_checks.py
"""Offline CPU checks shipped inside the notebook; no model/dataset downloads."""
import copy
import torch
from model import token_logps
from transformers import Phi3Config, Phi3ForCausalLM, Gemma3TextConfig, Gemma3ForCausalLM
from peft import LoraConfig, get_peft_model
from core import LagrangianController, FlipController, robust_flags
from io_utils import ROOT, read, file_hash


def check():
    torch.set_num_threads(1)
    assert file_hash(ROOT/'core.py') == read(ROOT/'SOURCE.json')['core_sha256']
    pi = LagrangianController('laconic', budgets=[100], max_len=4096, eta=.05, kp=4, two_sided=True)
    pi.ceiling = lambda k: 3
    pi.dual_update([110, 110], [0, 0])
    assert abs(pi.lambdas[0]-.405) < 1e-8
    ctrl = FlipController('nafc_replay', eps=.02, eta=10, repair_idx=[0, 1], held_idx=[2, 3])
    ctrl.update([True, False, False, False], [True]*4)
    assert ctrl.lam == 0  # Repair flags alone cannot activate the held controller.
    ctrl.update([True, False, True, False], [True]*4)
    assert ctrl.lam > 0
    assert robust_flags([[1, 0, 0, 0], [1, 1, 0, 0]]) == [True, False]
    configs = [
        (Phi3ForCausalLM, Phi3Config(vocab_size=13, hidden_size=16, intermediate_size=32,
            num_hidden_layers=2, num_attention_heads=2, num_key_value_heads=1,
            max_position_embeddings=128, original_max_position_embeddings=128,
            pad_token_id=0, bos_token_id=1, eos_token_id=12)),
        (Gemma3ForCausalLM, Gemma3TextConfig(vocab_size=13, hidden_size=16, intermediate_size=32,
            num_hidden_layers=2, num_attention_heads=2, num_key_value_heads=1, head_dim=8,
            max_position_embeddings=128, sliding_window=16, sliding_window_pattern=2,
            final_logit_softcapping=10, pad_token_id=0, bos_token_id=1, eos_token_id=12))]
    for model_class, config in configs:
        torch.manual_seed(1)
        a = get_peft_model(model_class(config), LoraConfig(r=2, lora_alpha=4, lora_dropout=0,
            target_modules='all-linear', task_type='CAUSAL_LM'))
        b = copy.deepcopy(a); b._pilot_chunked_head = True
        response = [i % 13 for i in range(65)]
        original, chunked = token_logps(a, [1, 2], response), token_logps(b, [1, 2], response)
        torch.testing.assert_close(original, chunked, rtol=1e-5, atol=1e-6)
        original.mean().backward(); chunked.mean().backward()
        for p, q in zip(a.parameters(), b.parameters()):
            if p.requires_grad:
                torch.testing.assert_close(p.grad, q.grad, rtol=1e-4, atol=1e-6)
    print('[CPU checks] unchanged PI/NAFC formulas and Phi/Gemma chunked-head values/gradients passed.', flush=True)


if __name__ == '__main__': check()


### `manual_artifacts.py`

In [ ]:
%%writefile /content/nafc_a100/manual_artifacts.py
"""Portable shared/seed ZIPs, with identity checks before merging local results."""
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
import zipfile
from io_utils import read, file_hash

MANIFEST = 'NAFC_BUNDLE.json'
REPORT_FILES = ('comparison.json', 'REPORT.md', 'overview.png', 'trajectory.png')


def pack(output, archive, seed=None):
    """Pack shared preparation, one seed, or the full notebook (seed='all')."""
    output, archive = Path(output).resolve(), Path(archive).resolve()
    identity = read(output/'shared'/'identity.json')
    folders = [output/'shared'] if seed in (None, 'all') else [output/f'seed_{seed}']
    if seed == 'all':
        folders += [output/f'seed_{value}' for value in identity['config']['seeds'] if (output/f'seed_{value}').exists()]
    if archive.is_relative_to(output) or any(not folder.is_dir() for folder in folders):
        raise ValueError('Save archives outside outputs; the selected results folder must exist.')
    prepared = output/'shared'/'prepared.json'
    manifest = {'format': 1, 'identity': identity, 'seed': seed,
                'prepared_sha256': file_hash(prepared) if prepared.exists() else None}
    if ((seed not in (None, 'all') or len(folders) > 1) and manifest['prepared_sha256'] is None):
        raise ValueError('Seed results need their frozen shared preparation.')
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(archive)+'.tmp')
    try:
        with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
            bundle.writestr(MANIFEST, json.dumps(manifest, ensure_ascii=False))
            paths = [path for folder in folders for path in folder.rglob('*')]
            if seed in (None, 'all'): paths += [output/name for name in REPORT_FILES if (output/name).exists()]
            for path in sorted(paths):
                if path.is_symlink():
                    raise ValueError('Result archives cannot contain symbolic links.')
                if path.is_file() and not path.name.endswith('.tmp'):
                    bundle.write(path, path.relative_to(output).as_posix())
        os.replace(temporary, archive)
    finally:
        temporary.unlink(missing_ok=True)
    return archive


def unpack(archives, output, config):
    output = Path(output).resolve(); output.mkdir(parents=True, exist_ok=True)
    files, identity, seeds = {}, None, []
    shared_identity = output/'shared'/'identity.json'
    if shared_identity.exists(): identity = read(shared_identity)
    for archive in map(Path, archives):
        with zipfile.ZipFile(archive) as bundle:
            if len(bundle.namelist()) != len(set(bundle.namelist())):
                raise ValueError('Duplicate archive members.')
            meta = json.loads(bundle.read(MANIFEST))
            if meta.get('format') != 1 or meta['identity']['config'] != config:
                raise ValueError('Archive model/configuration differs from this notebook.')
            if identity is not None and identity != meta['identity']:
                raise ValueError('Do not mix different models, environments or shared preparations.')
            identity = meta['identity']
            seed = meta['seed']
            if seed not in (None, 'all') and seed not in config['seeds']:
                raise ValueError('Archive contains an undeclared seed.')
            prefixes = {'shared'} if seed in (None, 'all') else {f'seed_{seed}'}
            if seed == 'all': prefixes |= {f'seed_{value}' for value in config['seeds']}
            if seed not in (None, 'all'): seeds.append(meta['prepared_sha256'])
            for info in bundle.infolist():
                if info.filename == MANIFEST: continue
                path = PurePosixPath(info.filename)
                if (info.orig_filename != info.filename or '\\' in info.filename or ':' in info.filename or path.is_absolute() or '..' in path.parts
                    or not path.parts or (path.parts[0] not in prefixes and not (seed in (None, 'all') and info.filename in REPORT_FILES))
                    or stat.S_ISLNK(info.external_attr >> 16)):
                    raise ValueError('Unsafe or unexpected archive path.')
                if info.is_dir(): continue
                if seed == 'all' and path.parts[0].startswith('seed_'):
                    seeds.append(meta['prepared_sha256'])
                target = output.joinpath(*path.parts).resolve()
                if not target.is_relative_to(output): raise ValueError('Unsafe archive destination.')
                with bundle.open(info) as stream:
                    h = hashlib.sha256()
                    for chunk in iter(lambda: stream.read(8*1024*1024), b''): h.update(chunk)
                digest = h.hexdigest()
                if info.filename in files and files[info.filename][2] != digest:
                    raise ValueError('Conflicting seed ZIPs; upload only the latest ZIP for each seed.')
                if target.exists() and file_hash(target) != digest:
                    raise ValueError('Existing results differ; restore the latest ZIPs into a fresh runtime.')
                files[info.filename] = (archive, info, digest)
    prepared = files.get('shared/prepared.json')
    prepared_hash = prepared[2] if prepared else file_hash(output/'shared'/'prepared.json') if (output/'shared'/'prepared.json').exists() else None
    if seeds and (prepared_hash is None or any(h != prepared_hash for h in seeds)):
        raise ValueError('Upload the matching shared ZIP together with seed ZIPs.')
    missing_bytes = sum(info.file_size for name, (_, info, _) in files.items() if not (output/name).exists())
    if missing_bytes > shutil.disk_usage(output).free:
        raise OSError('Not enough local space to restore these results.')
    for name, (archive, info, _) in files.items():
        target = output/name
        if target.exists(): continue
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = None
        try:
            with zipfile.ZipFile(archive) as bundle, bundle.open(info) as source:
                with tempfile.NamedTemporaryFile(dir=target.parent, suffix='.tmp', delete=False) as destination:
                    temporary = Path(destination.name)
                    shutil.copyfileobj(source, destination)
            os.replace(temporary, target)
        finally:
            if temporary is not None: temporary.unlink(missing_ok=True)
    return len(files)


### `config_colab.json`

In [ ]:
%%writefile /content/nafc_a100/config_colab.json
{
  "base_model": "microsoft/Phi-4-mini-instruct",
  "base_revision": "cfbefacb99257ffa30c83adab238a50856ac3083",
  "numina_revision": "1b05109f9e5c1ad06c0663519502416c30b300f8",
  "omni_revision": "40ba231d8f16e29ecd40e6407e2c8640145a8f62",
  "numina_sources": [
    "olympiads_ref",
    "olympiads"
  ],
  "seed": 42,
  "seeds": [
    42,
    43,
    44
  ],
  "split_seed": 42,
  "steps": 300,
  "prompts_per_step": 8,
  "group_size": 8,
  "train_count": 2000,
  "validation_count": 64,
  "anchor_pool": 2048,
  "anchor_count": 128,
  "anchor_minimum": 128,
  "anchor_selection_batch": 16,
  "initial_draws": 8,
  "eval_draws": 4,
  "generation_batch": 8,
  "max_new_tokens": 4096,
  "max_prompt_tokens": 4096,
  "lora_rank": 32,
  "learning_rate": 1e-05,
  "warmup_steps": 10,
  "compression_ratio": 0.7,
  "pi_eta": 0.05,
  "pi_kp": 4.0,
  "pi_lambda_max": 3.0,
  "audit_every": 25,
  "noise_margin": 0.02,
  "nafc_eta": 10.0,
  "nafc_lambda_max": 1.0,
  "replay_frac_max": 0.5,
  "validation_steps": [],
  "test_steps": [
    50,
    100,
    150,
    200,
    300
  ],
  "precision": "fp16",
  "loss_scale": 128,
  "chunked_head": true,
  "sample_progress": true,
  "checkpoint_every": 4,
  "test_do_sample": true,
  "test_draws": 4,
  "long_evaluation": true,
  "long_eval_tokens": 8192,
  "target_gpu": "A100",
  "test_count": 2000,
  "test_domain": "omni",
  "data_file": "colab_splits.json",
  "context_length": 131072
}


### `colab_splits.json`

In [ ]:
%%writefile /content/nafc_a100/colab_splits.json
{
  "settings": {
    "split_seed": 42,
    "train_count": 2000,
    "validation_count": 64,
    "anchor_pool": 2048,
    "numina_revision": "1b05109f9e5c1ad06c0663519502416c30b300f8",
    "omni_revision": "40ba231d8f16e29ecd40e6407e2c8640145a8f62",
    "numina_sources": [
      "olympiads_ref",
      "olympiads"
    ],
    "test_count": 2000
  },
  "splits": {
    "task_train": [
      {
        "id": "0df1e80f1473960fcc596f532187153673ba4983559a4729c72c19b8fec5b901",
        "domain": "numina",
        "question": "Let $0 \\leq a, b, c, d \\leq 2005$ be integers. What is the probability that $a b+c d$ is an even number?",
        "answer": "0.625",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171722,
        "difficulty": null
      },
      {
        "id": "d40b5a1900ec0ed00b4a71ce0a33148e9735fe6621d39f31ffb13406b84d8c90",
        "domain": "numina",
        "question": "4. In a certain company, there are 100 shareholders, and any 66 of them own no less than $50 \\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own?",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18609,
        "difficulty": null
      },
      {
        "id": "d2174b79fb3193e4b2f6c30bd8ef8f8fb238af7c8931b9c831e7d03d859b41c5",
        "domain": "numina",
        "question": "9.2. On the board in the laboratory, two numbers are written. Every day, the senior researcher Pyotr Ivanovich erases both numbers from the board and writes down their arithmetic mean and harmonic mean ${ }^{2}$. In the morning of the first day, the numbers 1 and 2 were written on the board.\n\nFind the product of the numbers written on the board in the evening of the 2016th day.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5712,
        "difficulty": null
      },
      {
        "id": "89d555273ff93f7241c554eac3557740a1efcd7cb522f6e3fe1da55522fdf0fc",
        "domain": "numina",
        "question": "Do the three empty fields in the image belong to such natural numbers so that the product of the three numbers on each side of the triangle is the same.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_cd0342d3a611cc15345bg-1.jpg?height=297&width=346&top_left_y=622&top_left_x=889)\n\nWhat is the smallest and what is the largest number that can be written in the gray-shaded field under this condition?",
        "answer": "1336",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76284,
        "difficulty": null
      },
      {
        "id": "4bc55b3f3ed0e5f2bb574f597f8b43d1ceb2c55852af19c0ad97b26dcbae0324",
        "domain": "numina",
        "question": "3+ [ Divisibility of numbers. General properties ]\n\nHumpty-Dumpty walks along a straight line, either taking 37 steps to the left or 47 steps to the right in a minute.\n\nWhat is the shortest time in which he can be one step to the right of the starting point?",
        "answer": "59",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68030,
        "difficulty": null
      },
      {
        "id": "de5b47edab8932b570f8d58fc75f67636b047b5548c78c3d9a5633ca037bfec5",
        "domain": "numina",
        "question": "[ Combinations and Permutations ]\n[ $\\underline{\\text { Directed Graphs }}$ ]\n\nIn the discussion, 15 deputies participated. Each of them, in their speech, criticized exactly $k$ of the remaining 14 deputies.\n\nFor what smallest $k$ can we assert that there will be two deputies who criticized each other?",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57345,
        "difficulty": null
      },
      {
        "id": "266d38a0f2c1af919c5e4a7a09c42017d7c205605f2cfad96904ea8999441a8f",
        "domain": "numina",
        "question": "71. A segment $AB$ of unit length, which is a chord of a sphere with radius 1, is positioned at an angle of $\\pi / 3$ to the diameter $CD$ of this sphere. The distance from the end $C$ of the diameter to the nearest end $A$ of the chord $AB$ is $\\sqrt{2}$. Determine the length of the segment $BD$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29780,
        "difficulty": null
      },
      {
        "id": "aa79ee2ab82c237a635d63a23c1d8446e22298a90cfdf304f84f1252a6c55a44",
        "domain": "numina",
        "question": "1.1. In a cross-country skiing race, Petya and Vasya started simultaneously. Vasya ran the entire race at a constant speed of 12 km/h. Petya ran the first half of the distance at a speed of 9 km/h and fell behind Vasya. What should Petya's speed be on the second half of the distance to catch up with Vasya and finish at the same time as his friend? Express your answer in km/h.",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17300,
        "difficulty": null
      },
      {
        "id": "b830058d56d98111ea25414125b676de6a266800bb0768ccbdee18aae2f6971c",
        "domain": "numina",
        "question": "Task 1 - 201211\nIn the following schema, a digit is to be entered in each empty field so that a correctly calculated division problem is formed. In particular, none of the multi-digit numbers in the completed schema may receive a leading digit of 0.\n\nProve that there is exactly one entry of digits in the empty fields that meets these requirements! Determine this entry!\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-2089.jpg?height=662&width=679&top_left_y=443&top_left_x=997)",
        "answer": "123976316",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 185986,
        "difficulty": null
      },
      {
        "id": "0cb26e341d59188d27ec772d9b115ae3a105aa14abb940aef2dbaa20887ad4bb",
        "domain": "numina",
        "question": "7. In a bag, there are two black and two white Go stones. Now, stones are taken from the bag, one at a time. If a white stone is drawn, it is kept. If a black stone is drawn, a white stone must be put back. If there are no white stones left to put back, the game ends. What is the probability of successfully taking all the stones? $\\qquad$",
        "answer": "\\frac{7}{18}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146025,
        "difficulty": null
      },
      {
        "id": "c31fdad4c31cd05ac3d7c8bd2584b7dde6f9b73644e6abbe815d3ef4574e69b8",
        "domain": "numina",
        "question": "2. A three-digit number $X$ was written with three different digits $A B C$. Four schoolchildren made the following statements. Petya: “The largest digit in the number $X$ is $B$”. Vasya: “$C=8$”. Tolya: “The largest digit is $C$”. Dima: “$C$ is the arithmetic mean of $A$ and $B$”. Find the number $X$, given that exactly one of the children is wrong.",
        "answer": "798",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 15035,
        "difficulty": null
      },
      {
        "id": "28443cefab2bce667dab6259a0f097e4c1d1d5027ea682537c7d404dff22e902",
        "domain": "numina",
        "question": "10. Let $m$ be the product of all positive integer divisors of 360,000 . Suppose the prime factors of $m$ are $p_{1}, p_{2}, \\ldots, p_{k}$, for some positive integer $k$, and $m=p_{1}^{e_{1}} p_{2}^{e_{2}} \\cdot \\ldots p_{k}^{e_{k}}$, for some positive integers $e_{1}, e_{2}, \\ldots, e_{k}$. Find $e_{1}+e_{2}+\\ldots+e_{k}$.",
        "answer": "630",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154320,
        "difficulty": null
      },
      {
        "id": "75bbae824896eceed8cf3fdfc1c4add55f4d599f07c27a16cf4e742600987efb",
        "domain": "numina",
        "question": "【Question 18】\nIf $a, b, c, d, e$ are consecutive positive integers, and $a<b<c<d<e, b+c+d$ is a perfect square, $a+b+c+d+e$ is a perfect cube, what is the minimum value of $c$?",
        "answer": "675",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 115355,
        "difficulty": null
      },
      {
        "id": "19d1534d9788f42f82ececdb35e8a27ec2950ffdc04c0642f958b549fe9e9fed",
        "domain": "numina",
        "question": "4. It is known that $x+y+z=2016$ and $\\frac{1}{x+y}+\\frac{1}{y+z}+\\frac{1}{z+x}=\\frac{1}{32}$. Find $\\frac{z}{x+y}+\\frac{x}{y+z}+\\frac{y}{z+x}$\n\nANSWER: 60.",
        "answer": "63",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6385,
        "difficulty": null
      },
      {
        "id": "c85be59a2cac309e5ee9b5fd68070429c19325bc4360eab7109cf970e65e794e",
        "domain": "numina",
        "question": "1. On the Island of Knights and Liars, there live knights who always tell the truth and liars who always lie. One day, five residents of this island were asked one by one how many knights were among them.\n\n- One, - answered the first.\n- Two, - answered the second.\n- Three, - answered the third.\n- Don't believe them, they are all liars, - said the fourth.\n- You are the liar! - said the fifth to the fourth.\n\nHow many knights were there in reality?\n\nAnswer: 2.",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 6193,
        "difficulty": null
      },
      {
        "id": "515f133ad07c5ea377c25710b4404f00885a42564a23f9d8a3d2280b3d2ab736",
        "domain": "numina",
        "question": "Task 6.4. On a computer keyboard, the key with the digit 1 is not working. For example, if you try to type the number 1231234, only the number 23234 will be printed.\n\nSasha tried to type an 8-digit number, but only 202020 was printed. How many 8-digit numbers meet this condition?",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16056,
        "difficulty": null
      },
      {
        "id": "a410eb23e6a939e2849e80c9ec93730f2703a8357c5ecec5a90a3c2814c807f9",
        "domain": "numina",
        "question": "A and B are shooting at a shooting range, but they only have one six-chambered revolver with one bullet. Therefore, they agreed to take turns randomly spinning the cylinder and shooting. A starts. Find the probability that the shot will occur when the revolver is with A.",
        "answer": "\\frac{6}{11}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55841,
        "difficulty": null
      },
      {
        "id": "583b97237a8eb4e256f1ed79e1907f4a5dee2bde8c9ddeada54270a24b955ea6",
        "domain": "numina",
        "question": "6,7\n\nThe population of China is one billion people. It would seem that on a map of China with a scale of $1: 1000000$ (1 cm $: 10$ km), one million times fewer people could fit, compared to the entire territory of the country. However, in reality, not only 1000, but even 100 people would not be able to fit on this map. Can you explain this contradiction?",
        "answer": "1000000^2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 61958,
        "difficulty": null
      },
      {
        "id": "da4bdac0728dd971c77d2df22926f2a522cb8960f6dc3c0c47197186c833a940",
        "domain": "numina",
        "question": "11. In the sequence $\\left\\{a_{n}\\right\\}$, $a_{1}, a_{2}$ are given non-zero integers, $a_{n+2}=\\left|a_{n+1}-a_{n}\\right|$.\n(1) If $a_{16}=4, a_{17}=1$, find $a_{2018}$;\n(2) Prove: From $\\left\\{a_{n}\\right\\}$, it is always possible to select infinitely many terms to form two different constant subsequences.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136340,
        "difficulty": null
      },
      {
        "id": "c04d24610ea020b196ab32cc0537009e35263119a461a85ea5ccd9700718f9c1",
        "domain": "numina",
        "question": "18. Find the largest integer $n$ such that $n$ is a divisor of $a^{5}-a$ for all integers $a$.",
        "answer": "30",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160207,
        "difficulty": null
      },
      {
        "id": "7482eb168165d33857a9676fe4052eb75a3a54a734676d8ba1de6380bf16ce90",
        "domain": "numina",
        "question": "14.1. $[9.1$ (15 points), 10.1 (15 points)] Calculate\n\n$$\n\\sqrt{1+2 \\sqrt{1+3 \\sqrt{1+\\ldots+2017 \\sqrt{1+2018 \\cdot 2020}}}}\n$$\n\nAnswer: 3.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5890,
        "difficulty": null
      },
      {
        "id": "92dc117837ae23a99a2545f0a01e36cf3b77ca26215f6ae7c609b73e6d0cb459",
        "domain": "numina",
        "question": "6. Nathaniel and Obediah play a game in which they take turns rolling a fair six-sided die and keep a running tally of the sum of the results of all rolls made. A player wins if, after he rolls, the number on the running tally is a multiple of 7 . Play continues until either player wins, or else indefinitely. If Nathaniel goes first, determine the probability that he ends up winning.",
        "answer": "\\frac{5}{11}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151287,
        "difficulty": null
      },
      {
        "id": "24205b9a2b7e36be301943ae471888449d84df5901ea9009903cb11074f06232",
        "domain": "numina",
        "question": "13.328. A fishing brigade planned to catch 1800 tons of fish within a certain period. For $1 / 3$ of this period, there was a storm, as a result of which the planned daily target was underachieved by 20 tons. However, in the remaining days, the brigade managed to catch 20 tons more than the daily norm, and the planned target was completed one day ahead of schedule. How many tons of fish were planned to be caught daily?",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49414,
        "difficulty": null
      },
      {
        "id": "cc87f86c520dc9d46f78a94e422c154bb5f0471a934b8a6173b4f32517edc269",
        "domain": "numina",
        "question": "How long would it take for the Earth to complete its axial rotation so that the centrifugal force at the equator would cancel out the weight of objects? The question also needs to be solved by trial and error: at the current rotational speed, at what distance from the Earth would the centrifugal force be equal to the gravitational force?",
        "answer": "5062",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 176742,
        "difficulty": null
      },
      {
        "id": "e3e62413e72e1f4bae77cd191d70eca9c042d0459a0aa0f2f9eef45f79ce49f0",
        "domain": "numina",
        "question": "What is the sum of the $x$-intercept of the line with equation $20 x+16 y-40=0$ and the $y$-intercept of the line with equation $20 x+16 y-64=0$ ?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194436,
        "difficulty": null
      },
      {
        "id": "29d046ef651ae2243a9178f6e586b67b6e7559dc7aa035274f3b9e3d51ad1035",
        "domain": "numina",
        "question": "8. (5 points) Teacher Wang is distributing bananas to the children. If each child gets 3 bananas, there will be 4 bananas left; if each child gets 4 bananas, there will be 3 bananas short. How many children are there $\\qquad$?",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145914,
        "difficulty": null
      },
      {
        "id": "907e3835229ecf1605ecb5f608a9960b3f45f2d9fc48ceed61b871956598958a",
        "domain": "numina",
        "question": "$$\n\\begin{array}{l}\n\\text { 9. In } \\triangle A B C, \\angle A<\\angle B<\\angle C, \\\\\n\\frac{\\sin A+\\sin B+\\sin C}{\\cos A+\\cos B+\\cos C}=\\sqrt{3} .\n\\end{array}\n$$\n\nThen $\\angle B=$",
        "answer": "\\frac{\\pi}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114735,
        "difficulty": null
      },
      {
        "id": "a20915169b7886ebb52c4bfd048b45dbfccc98e7f7157e143044d0e6c3e1f8ad",
        "domain": "numina",
        "question": "Golenischeva-Kumuzova T.I.\n\nYura has a calculator that allows multiplying a number by 3, adding 3 to a number, or (if the number is divisible by 3) dividing the number by 3. How can Yura use this calculator to get from the number 1 to the number 11?",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56349,
        "difficulty": null
      },
      {
        "id": "c00c75ca1a731848e25140f41b322c41e6f43f1b7b1df7ac8d768756185bab7f",
        "domain": "numina",
        "question": "6. On the interval $\\left[\\frac{1}{2}, 2\\right]$, the function $f(x)=x^{2}+p x+q$ and $g(x)=2 x+\\frac{1}{x^{2}}$ achieve the same minimum value at the same point. The maximum value of $f(x)$ on $\\left[\\frac{1}{2}, 2\\right]$ is $\\qquad$ .",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122227,
        "difficulty": null
      },
      {
        "id": "592dbbad3bcd9dfcca03f5d7e130f5aba06d358530c94b7d64c8c2c357030967",
        "domain": "numina",
        "question": "16. Consider the function $f$ defined by\n$$\nf(x)=1+\\frac{2}{x} .\n$$\n\nFind the roots of the equation\n$$\n(\\underbrace{f \\circ f \\circ \\cdots \\circ f}_{10 \\text { times }})(x)=x,\n$$\nwhere \"o\" denotes composition of functions.",
        "answer": "-12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 154041,
        "difficulty": null
      },
      {
        "id": "248ce4c0e78ee1a9b6b1380d6e9059e149fa99a46e98fa7e7a96b83448c474f4",
        "domain": "numina",
        "question": "I am thinking of a five-digit number composed of even digits. If I swap the digit in the third position $\\mathrm{s}$ with any other, the number decreases. Furthermore, I will reveal that the first digit is double the last, and the second digit is double the second to last.\n\nWhat number am I thinking of?\n\n(M. Mach)\n\nHint. Can you compare the third digit with the others?",
        "answer": "88644",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76691,
        "difficulty": null
      },
      {
        "id": "30eaca3574d960044b6cd98edc4661fa2e054a576943bff2288d94d8e47df154",
        "domain": "numina",
        "question": "18. For each natural $n$, which is not a perfect square, the number of values of the variable $x$ is calculated, for which both numbers $x+\\sqrt{n}$ and $x^{2}+param1 \\cdot \\sqrt{n}$ are natural numbers less than param2. Find the total number of such values of $x$.\n\n| param1 | param2 | answer |\n| :---: | :---: | :---: |\n| 18 | 200 |  |\n| 20 | 210 |  |\n| 22 | 250 |  |\n| 24 | 300 |  |\n| 26 | 330 |  |",
        "answer": "108",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 3055,
        "difficulty": null
      },
      {
        "id": "95ac13ee46867916a9df994961e729e3d57f4949c260f8cf7dae4e6051b84123",
        "domain": "numina",
        "question": "122. Get rid of the irrationality in the denominator of the fraction\n\n$$\n\\frac{2+\\sqrt{3}}{\\sqrt{6}-\\sqrt{3}+\\sqrt{2}-1} .\n$$",
        "answer": "\\frac{1}{2}(\\sqrt{3}+1)(\\sqrt{2}+1)",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36046,
        "difficulty": null
      },
      {
        "id": "52b11f7e55cb356908ec109a004035a98f3346f4aed583ed593d254f13fc4f9a",
        "domain": "numina",
        "question": "Suppose that\n\n$$\n\\begin{aligned}\nM & =1^{5}+2^{4} \\times 3^{3}-4^{2} \\div 5^{1} \\\\\nN & =1^{5}-2^{4} \\times 3^{3}+4^{2} \\div 5^{1}\n\\end{aligned}\n$$\n\nWhat is the value of $M+N$ ?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194185,
        "difficulty": null
      },
      {
        "id": "1bc280057e02b0f3da21b32c0de5bda95fc3fcc4a7453ace18c07c8578b43bc0",
        "domain": "numina",
        "question": "11. (20 points) Find the largest real number $k$ such that for any triangle with sides $a$, $b$, and $c$, the following inequality holds:\n$$\n\\frac{b c}{b+c-a}+\\frac{a c}{a+c-b}+\\frac{a b}{a+b-c} \\geqslant k(a+b+c) .\n$$",
        "answer": "1",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 115502,
        "difficulty": null
      },
      {
        "id": "021e105ab0270d922c8b712bbad3c79ab42e83af8ec0a7e00e414c815a06572a",
        "domain": "numina",
        "question": "5. The line segment connecting two points on a sphere is called a chord of the sphere. For a sphere with a radius of 4, the lengths of two chords $AB$ and $CD$ are $2\\sqrt{7}$ and $4\\sqrt{3}$, respectively. $M$ and $N$ are the midpoints of $AB$ and $CD$, respectively. The endpoints of each pair of chords can move on the surface of the sphere. There are the following four propositions:\n(1) Chords $AB$ and $CD$ may intersect at point $M$\n(2) Chords $AB$ and $CD$ may intersect at point $N$\n(3) The maximum value of $MN$ is 5\n(4) The minimum value of $MN$ is 1\nAmong these, the true propositions are",
        "answer": "(1)(3)(4)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 95472,
        "difficulty": null
      },
      {
        "id": "3e06a4f9d4d33412798ff1e49da8acb668aff41aad7ac45e6e1eca0e6f571abd",
        "domain": "numina",
        "question": "The average age of the Kebulov family, consisting of a father, a mother, and several children, is 18 years. When the father, who is 38 years old, is not included, the average age of the family is 14 years. How many children do the Kebulovs have?\n\n(L. Hozová)",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76655,
        "difficulty": null
      },
      {
        "id": "70ffcac08e08bd89cac9c841bad1ef0bf2dd445c7f27b0575a7aed585046bf97",
        "domain": "numina",
        "question": "44. Find a two-digit number that is equal to three times the product of its digits.",
        "answer": "2415",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 34775,
        "difficulty": null
      },
      {
        "id": "f0b8c4562931551834a977f9dba51f145473140f51dcff703ce47f1b017d9d59",
        "domain": "numina",
        "question": "Kosukhin O.n.\n\nSasha found that there were exactly $n$ working digit buttons left on the calculator. It turned out that any natural number from 1 to 99999999 can either be entered using only the working buttons, or obtained as the sum of two natural numbers, each of which can be entered using only the working buttons. What is the smallest $n$ for which this is possible?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 54154,
        "difficulty": null
      },
      {
        "id": "c9e69bbf5bb8c80ed00466bcbbbd62560e0af03b7bda06cc122107d4ca817fe1",
        "domain": "numina",
        "question": "$30.11 \\times 12 \\times 13 \\times 14 \\times 15 \\times 16 \\times 17 \\times 18$, the unit digit of the product is",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94587,
        "difficulty": null
      },
      {
        "id": "821c6e5de22677f10321b038a0b5d3a6bdc31d3248522645b6b5335b56b51a15",
        "domain": "numina",
        "question": "Variant 9.2.3. On the sides $AB$ and $AD$ of rectangle $ABCD$, points $M$ and $N$ are marked, respectively. It is known that $AN=3$, $NC=39$, $AM=10$, $MB=5$.\n\n(a) (1 point) Find the area of rectangle $ABCD$.\n\n(b) (3 points) Find the area of triangle $MNC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f4981519479532effc1g-10.jpg?height=425&width=614&top_left_y=389&top_left_x=420)",
        "answer": "585",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17257,
        "difficulty": null
      },
      {
        "id": "2fc37516ebef3ff4a94f08af33f1feb3397a8bf45e2ea0c056b24eff7f022b9e",
        "domain": "numina",
        "question": "# 3. CONDITION\n\nGiven a right triangle $A B C$ with legs $A C=3$ and $B C=4$. Construct triangle $A_{1} B_{1} C_{1}$ by sequentially moving point $A$ a certain distance parallel to segment $B C$ (point $A_{1}$ ), then point $B-$ parallel to segment $A_{1} C$ (point $B_{1}$ ), and finally point $C$ - parallel to segment $A_{1} B_{1}$ (point $C_{1}$ ). What is the length of segment $B_{1} C_{1}$, if it turns out that angle $A_{1} B_{1} C_{1}$ is a right angle and $A_{1} B_{1}=1$?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15920,
        "difficulty": null
      },
      {
        "id": "83f344e81f03af023996e9e78b3c7ec0e66a9808e0e236e60c1739a29d7e2051",
        "domain": "numina",
        "question": "7. Given the sets\n$$\n\\begin{array}{l}\nA=\\left\\{(x, y) \\mid x=m, y=-3 m+2, m \\in \\mathbf{Z}_{+}\\right\\}, \\\\\nB=\\left\\{(x, y) \\mid x=n, y=a\\left(a^{2}-n+1\\right), n \\in \\mathbf{Z}_{+}\\right\\} .\n\\end{array}\n$$\n\nThen the number of integers $a$ such that $A \\cap B \\neq \\varnothing$ is $\\qquad$.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105031,
        "difficulty": null
      },
      {
        "id": "eb0e3eb7e4d34211729cecf2636381150bb8f4961da201ae647f140d30fef2f9",
        "domain": "numina",
        "question": "9.5. From the digits $1,2,3,4,5,6,7,8,9$, nine (not necessarily distinct) nine-digit numbers are formed; each digit is used exactly once in each number. What is the maximum number of zeros that the sum of these nine numbers can end with?\n\n(N. Agakhanov)",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16511,
        "difficulty": null
      },
      {
        "id": "6fa1a390c7f6a383297b8b2c5ff7130187d419df73617eec863343b534788caa",
        "domain": "numina",
        "question": "【Example 2】Using 4 different colors to color the 4 sides of a small square wooden block, with each side painted a different color, how many different coloring patterns can there be?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 99877,
        "difficulty": null
      },
      {
        "id": "6ee8b504f7fa99ead77d82af11f30eda70d4c916b41255210c092299ad40c57b",
        "domain": "numina",
        "question": "171 If three integers $a, b, c(a \\neq 0)$ make the equation $a x^{2}+b x+c=0$ have two roots $a$ and $b$, then $a+b+c=$",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96348,
        "difficulty": null
      },
      {
        "id": "5fe4cc3a0ec007b3b2870e6368364fb0645c86b010746e52971bbf132f3249d1",
        "domain": "numina",
        "question": "23. In the figure below, $\\mathrm{ADE}$ is a triangle with $\\angle \\mathrm{AED}=120^{\\circ}$, and $\\mathrm{B}$ and $\\mathrm{C}$ are points on side $\\mathrm{AD}$ such that $\\mathrm{BCE}$ is an equilateral triangle. If $\\mathrm{AB}=4 \\mathrm{~cm}, \\mathrm{CD}=16 \\mathrm{~cm}$ and $\\mathrm{BC}=x \\mathrm{~cm}$, find the value of $x$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160388,
        "difficulty": null
      },
      {
        "id": "90eac8b78d2bb30dc32ad2c744bc687b36fc048a02a7d52398ea337f3d665e99",
        "domain": "numina",
        "question": "The figure shows a rectangle $A B C D$ with side $\\overline{A D}=4$ and where point $M$ is the midpoint of segment $A B$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-19.jpg?height=508&width=708&top_left_y=500&top_left_x=777)\n\nKnowing that segments $A C$ and $D M$ are orthogonal, calculate the area of rectangle $A B C D$.\n\n#",
        "answer": "16\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73882,
        "difficulty": null
      },
      {
        "id": "32db5981ee004d06f17185be48881c77d998f1965087a776882349375823f163",
        "domain": "numina",
        "question": "3. Quadrilateral $ABCD$ is inscribed in circle $\\omega$, the center of which lies on side $AB$. Circle $\\omega_{1}$ is externally tangent to circle $\\omega$ at point $C$. Circle $\\omega_{2}$ is tangent to circles $\\omega$ and $\\omega_{1}$ at points $D$ and $E$ respectively. Line $BD$ intersects circle $\\omega_{2}$ again at point $P$, and line $AC$ intersects circle $\\omega_{1}$ again at point $Q$. Find the angle $PEQ$.",
        "answer": "180",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8203,
        "difficulty": null
      },
      {
        "id": "10f6163c4c23d1fe7faf22f3d7bdbe0572e82d2d3ea5648feb59d7556ef4d85c",
        "domain": "numina",
        "question": "4. (6 points) The sum of the largest natural number less than 100 and the smallest natural number greater than 300, is the $\\qquad$ times of the largest natural number not greater than 200.\n\n\n\n保留了原文的格式和换行，翻译结果如上。注意，原文中的 $\\qquad$ 用于表示空格或留白，翻译时也保持不变。",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124779,
        "difficulty": null
      },
      {
        "id": "3636da103725d1653948bc16af5b5c7b9e46b5f1bac84ac2c432ae6ff2df8da2",
        "domain": "numina",
        "question": "Example 2 A deck of cards consists of 52 cards, with 13 cards of each of the four suits: \"Diamonds,\" \"Clubs,\" \"Hearts,\" and \"Spades,\" numbered $2,3, \\cdots, 10, \\mathrm{~J}, \\mathrm{Q}, \\mathrm{K}, \\mathrm{A}$, respectively. Two cards of the same suit and adjacent numbers are called \"consecutive suited cards,\" and $\\mathrm{A}$ can be considered as 1, making it consecutive with 2. Determine the number of ways to draw 13 cards from this deck such that each number appears and no two cards form a consecutive suited pair.",
        "answer": "3^{13}-3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143920,
        "difficulty": null
      },
      {
        "id": "33bf1212e56b373632cfed70cbecba1a681d49749833c880f82db3e9032a468a",
        "domain": "numina",
        "question": "8. (4 points) Given a triangle $ABC$ with sides $AB=14$, $BC=13$, and $AC=15$. On side $AB$, point $K$ is marked; on side $AC$, point $L$ is marked; and on side $BC$, point $N$ is marked. It is known that $AK=\\frac{15}{14}$, $BN=9$, and $AL=1$. A line parallel to $KL$ is drawn through point $N$, intersecting side $AC$ at point $M$. Find the area of quadrilateral $KLMN$.",
        "answer": "30\\frac{1013}{1183}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21028,
        "difficulty": null
      },
      {
        "id": "6db394a4d85aab74079ac2c75147a65584ca159bdccf45954cd4c9b1ddd9eae2",
        "domain": "numina",
        "question": "Example 6 Find the smallest positive period of the function $y=4 \\sin \\left(3 x+\\frac{\\pi}{4}\\right)+3 \\cos \\left(3 x+\\frac{\\pi}{4}\\right)$.",
        "answer": "\\frac{2\\pi}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104706,
        "difficulty": null
      },
      {
        "id": "a80e3f61ec60dcf375bd0c222e01f4b09d7c6ca5994a27ecb67b7fcd89ffe143",
        "domain": "numina",
        "question": "# Task 10.5\n\nThe field is a $41 \\times 41$ grid, in one of the cells of which a tank is hidden. The fighter aircraft shoots one cell per shot. If a hit occurs, the tank moves to an adjacent cell along a side; if not, it remains in place. After each shot, the pilot does not know whether a hit occurred. To destroy the tank, it must be hit twice. What is the minimum number of shots required to guarantee that the tank is destroyed?\n\n## Points 7\n\n#",
        "answer": "2521",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13710,
        "difficulty": null
      },
      {
        "id": "c27b337e3820c620eda38e4e6acac9a72e2842577888d43b817c2df69572af7d",
        "domain": "numina",
        "question": "60th Putnam 1999 Problem A5 Find a constant k such that for any polynomial f(x) of degree 1999, we have |f(0)| ≤ k ∫ -1 1 |f(x)| dx.",
        "answer": "(1/3000)^{2000}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 147947,
        "difficulty": null
      },
      {
        "id": "f6fea51a777276f30ef261406b94bc45be8817962377bd227f9e676506c6e966",
        "domain": "numina",
        "question": "Opgave 3. Gegeven is een onbekende rij $a_{1}, a_{2}, a_{3}, \\ldots$ van gehele getallen die voldoet aan de volgende eigenschap: voor elk priemgetal $p$ en elk positief geheel getal $k$ geldt\n\n$$\na_{p k+1}=p a_{k}-3 a_{p}+13\n$$\n\nBepaal alle mogelijke waarden van $a_{2013}$.",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85298,
        "difficulty": null
      },
      {
        "id": "c126da39d46034f11cb4a25c484776153ef3c76e9de563be037983da44ce34dd",
        "domain": "numina",
        "question": "In how many ways can an $n \\times n$ table be filled with positive integers from 1 to $n^{2}$ such that the difference between numbers written in adjacent cells is not greater than $(n+1)$? (Two cells are adjacent if they share an edge or a vertex.)",
        "answer": "32",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 168137,
        "difficulty": null
      },
      {
        "id": "366bd1a7e49dd57ee8288a9c01ecbf16fd692ad1ac0a46098dd2e6fb24b633f5",
        "domain": "numina",
        "question": "B1. The product of two consecutive natural numbers after the number $n$ is 600 greater than the product of two consecutive natural numbers before the number $n$. Determine the number $n$. Write down the solution.",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163893,
        "difficulty": null
      },
      {
        "id": "c7cb7358d738638b9a6e6f44d30ce751d45255c32ffd71bb819f9f16cbc319d2",
        "domain": "numina",
        "question": "11. In $\\triangle A B C$, it is known that $A=\\frac{\\pi}{3}, \\frac{A B}{A C}=\\frac{8}{5}$, and the inradius $r=2 \\sqrt{3}$. Then the area of this triangle is",
        "answer": "40\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 100505,
        "difficulty": null
      },
      {
        "id": "3f2d094c1c357c17fb0fcf67af06128d865162922e65585dc02a30a6ae2ea3fa",
        "domain": "numina",
        "question": "G1.2 Let $x=\\frac{1-\\sqrt{3}}{1+\\sqrt{3}}$ and $y=\\frac{1+\\sqrt{3}}{1-\\sqrt{3}}$. If $b=2 x^{2}-3 x y+2 y^{2}$, find the value of $b$.",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152909,
        "difficulty": null
      },
      {
        "id": "43befab6b031eedf46e0aa253fe456a04b60e0381f35aaeab7a4ac16150bbfc2",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n$\\lim _{n \\rightarrow \\infty} \\sqrt{n^{3}+8}\\left(\\sqrt{n^{3}+2}-\\sqrt{n^{3}-1}\\right)$",
        "answer": "\\frac{3}{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47477,
        "difficulty": null
      },
      {
        "id": "520450df7bf0547b348121819753ce55e076309160cb3a470cf0dcdae148c02f",
        "domain": "numina",
        "question": "11. II 5 (BUL 1) ${ }^{\\mathrm{IMO} 4}$ Consider a partition of an $8 \\times 8$ chessboard into $p$ rectangles whose interiors are disjoint such that each of them has an equal number of white and black cells. Assume that $a_{1}<a_{2}<\\cdots<a_{p}$, where $a_{i}$ denotes the number of white cells in the $i$ th rectangle. Find the maximal $p$ for which such a partition is possible and for that $p$ determine all possible corresponding sequences $a_{1}, a_{2}, \\ldots, a_{p}$.",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 196721,
        "difficulty": null
      },
      {
        "id": "29c6777e8e238e5d36c703c4493c164b37fdcb0be0ad639abf7b7faf9e7d7d8a",
        "domain": "numina",
        "question": "7-7. The recruits stood in a row, facing the same direction. Among them were three brothers: Peter, Nikolai, and Denis. In front of Peter were 50 people, in front of Nikolai 100, and in front of Denis 170. On the command \"About face!\" everyone turned to face the opposite direction. As a result, it turned out that in front of one of the brothers there are now four times as many people as in front of another. How many recruits could there be, including the brothers? List all possible options.",
        "answer": "211",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12366,
        "difficulty": null
      },
      {
        "id": "49c633a98f824b6341f800fef036ea0f3dddba48a5f248c8957db398dc9f348d",
        "domain": "numina",
        "question": "2. Let $x_{1}$ and $x_{2}$ be the largest roots of the polynomials\n\n$$\n\\begin{gathered}\nf(x)=1-x-4 x^{2}+x^{4} \\\\\n\\text { and } \\\\\ng(x)=16-8 x-16 x^{2}+x^{4}\n\\end{gathered}\n$$\n\nrespectively. Find $\\frac{x_{2}}{x_{1}}$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8602,
        "difficulty": null
      },
      {
        "id": "1d1aedcaa29971ac9c9fcf1dd1b31255ecf1c782feb51b2d9e46efba543e9eb6",
        "domain": "numina",
        "question": "## 126. Math Puzzle 11/75\n\nA gardener sold half of all his apples and half an apple to the first buyer, half of the remaining apples and another half an apple to the second buyer, half of the remaining apples and half an apple to the third buyer, and so on.\n\nTo the seventh buyer, he sold half of the remaining apples and another half an apple. Then he had none left.\n\nHow many apples did the gardener have at the beginning?",
        "answer": "127",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181805,
        "difficulty": null
      },
      {
        "id": "e6d598236695e4f0a0ee7b724ad576320db102fd49db360be03b5363b0bbf8cb",
        "domain": "numina",
        "question": "$[$ [extension of a tetrahedron to a parallelepiped]\n\nSegment $A B(A B=1)$, being a chord of a sphere with radius 1, is positioned at an angle of $60^{\\circ}$ to the diameter $C D$ of this sphere. The distance from the end $C$ of the diameter to the nearest end $A$ of the chord $A B$ is $\\sqrt{2}$. Find $B D$.\n\n#",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59496,
        "difficulty": null
      },
      {
        "id": "cb2f74cbd1b71dffaca0eb17f3087447d688ad757d42ce1fd07ed10096ffd291",
        "domain": "numina",
        "question": "(Moscow 2000, exercise A1) Let $x$ and $y$ be two different real numbers such that $x^{2}-2000 x=y^{2}-2000 y$. What is the value of $x+y$?",
        "answer": "2000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 82860,
        "difficulty": null
      },
      {
        "id": "d54cd0f4476083f7757fad2d51de4d919a7e03e15d393e7acac041b118c1dc0b",
        "domain": "numina",
        "question": "2. For natural numbers $a$ and $b$, it is known that $5a-1$ is divisible by $b$, $a-10$ is divisible by $b$, but $3a+5$ is not divisible by $b$. What values can the number $b$ take?",
        "answer": "49",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13843,
        "difficulty": null
      },
      {
        "id": "cf674a0d4d409cd0ee382a2a6d5d42adccb54962d5f3fafa8c2625591809c2dc",
        "domain": "numina",
        "question": "[ Law of Cosines ]\n\nFind the sum of the squares of the distances from a point $M$, taken on the diameter of a certain circle, to the ends of any chord parallel to this diameter, if the radius of the circle is $R$, and the distance from the point $M$ to the center of the circle is $a$.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62022,
        "difficulty": null
      },
      {
        "id": "8fceaeb241265f9b4be343949bacd07ff9ee0745c0cca015bbfdd93b135c6565",
        "domain": "numina",
        "question": "4-1. Katya attached a square with a perimeter of 40 cm to a square with a perimeter of 100 cm as shown in the figure. What is the perimeter of the resulting figure in centimeters?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_535cfcd84333341737d0g-01.jpg?height=281&width=374&top_left_y=676&top_left_x=844)",
        "answer": "120",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17120,
        "difficulty": null
      },
      {
        "id": "d1bf03dc6be685ec5852599cf877057a3e7ee0f87bc58fbbcd71ad192cbd0920",
        "domain": "numina",
        "question": "31. Given $\\overline{A B C D E F} \\times B=\\overline{E F A B C D}$, the same letter represents the same digit, and different letters represent different digits. Then the possible cases for $\\overline{A B C D E F}$ are $\\qquad$ kinds.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105080,
        "difficulty": null
      },
      {
        "id": "94e0745e1dac40ecb8fa00e3ee72b4f2e117c1fd32c26b5d08da774ead5055d9",
        "domain": "numina",
        "question": "In the homework for calculating the value of the expression\n\n$$\n2-3+4-5+6-7+8-9+10=\n$$\n\nRadek forgot to write two parentheses, so he got a result that was 18 greater than what he would have obtained if he had written the parentheses. Add the parentheses in two ways and write down what number Radek got and what number he should have gotten.\n\n(M. Dillingerová)",
        "answer": "-12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76671,
        "difficulty": null
      },
      {
        "id": "98b130ce48a140ceb0f125197d1d1abafa12b40063799685b9c063c795db3ad6",
        "domain": "numina",
        "question": "7. Let $\\sin \\alpha+\\sin \\beta=\\frac{4}{5} \\sqrt{2}, \\cos \\alpha+\\cos \\beta=\\frac{4}{5} \\sqrt{3}$, then $\\tan \\alpha+\\tan \\beta=$",
        "answer": "\\sqrt{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123259,
        "difficulty": null
      },
      {
        "id": "4223d3c86d4c3b2d9b26548ff76019de8ae4668654ff97015db8199490e2197c",
        "domain": "numina",
        "question": "The wall clock in the kitchen is 2 seconds slow every day, and the antique clock in the room is 15 seconds fast every day. On Sunday at noon, the wall clock shows 12:01, and the antique clock shows 11:59. When during the week will the sum of the squares of the differences between the times shown by the clocks and the actual time be the smallest?",
        "answer": "\\frac{1020}{229}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170459,
        "difficulty": null
      },
      {
        "id": "750ab5f3f39b35879b043d60403c3c89fe422cad10b6567d997a8d1bc966b16a",
        "domain": "numina",
        "question": "7. Given a tetrahedron \\(ABCD\\) with a volume of \\(40\\), \\(AB = 7\\), \\(BC = 8\\), \\(AC = 9\\). If the projection of vertex \\(D\\) onto the plane \\(ABC\\) is exactly the incenter \\(H\\) of \\(\\triangle ABC\\), then the surface area of the tetrahedron is\n\n---\n\nThe translation preserves the original text's formatting and line breaks.",
        "answer": "60+12\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 101340,
        "difficulty": null
      },
      {
        "id": "d6856b14172649305eeeefb72e00845428aa89f8b616c28d8fb9b4997504843d",
        "domain": "numina",
        "question": "180. Find the sides of a rectangle whose area is numerically equal to the sum of its perimeter and diagonal, and the base is three times the height.\n\n32",
        "answer": "8+\\sqrt{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 37817,
        "difficulty": null
      },
      {
        "id": "5c92f3a98def0312012c16ebc2e4aacf9d9d100cd7367ee482c58a154b66d056",
        "domain": "numina",
        "question": "7. The distances from the space diagonal of a rectangular prism to the three edges that do not intersect it are $2 \\sqrt{5}, \\frac{30}{\\sqrt{13}}, \\frac{15}{\\sqrt{10}}$, then the volume of the rectangular prism is $\\qquad$",
        "answer": "750",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102660,
        "difficulty": null
      },
      {
        "id": "30c0378d4353a44109a6049530ad5eb7fddd12302168b117c9956dcddbe26220",
        "domain": "numina",
        "question": "4. Given non-negative real numbers $x, y, z$ satisfy $\\frac{x}{2}+1=2-\\frac{y}{3}=\\frac{z}{4}-3$, then the minimum value of $2 x+3 y+4 z$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "73",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 98940,
        "difficulty": null
      },
      {
        "id": "3a5f8e37d14186478c12bd050b7e4b79d57d607645702731388c73fe86c6bc82",
        "domain": "numina",
        "question": "$3.438 \\sin 84^{\\circ} \\cdot \\sin 24^{\\circ} \\cdot \\sin 48^{\\circ} \\cdot \\sin 12^{\\circ}=\\frac{1}{16}$.",
        "answer": "\\frac{1}{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51872,
        "difficulty": null
      },
      {
        "id": "8508c9d95f56dd07bd099a144d4a5544dc4f1b062784843d2320c4c34828a784",
        "domain": "numina",
        "question": "## Task 1 - V10821\n\nIn an LPG, potatoes are laid out using a four-row laying machine. With this modern machine, 3 cooperative farmers can lay out 5 hectares of potatoes in 10 hours.\n\nIn the past, when farmers had to lay out the potatoes by hand, one farmer could manage 0.5 hectares in 8 hours. Additionally, he needed 2 hours for loosening the soil and covering the potatoes.\n\nHow many hours of work (labor hours) are required for 1 hectare\n\na) with the potato laying machine,\n\nb) with manual labor?\n\nc) Compare the figures by calculating percentages.",
        "answer": "30",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 183906,
        "difficulty": null
      },
      {
        "id": "5d4a3a040a0e1c47fa5e01ae179d1da118680ceae96df28b5c93b5cc1fca4844",
        "domain": "numina",
        "question": "Every term of a sequence, starting from the second, is equal to the sum of the previous term and the sum of its digits. The first elements of the sequence are\n\n$$\n1,2,4,8,16,23,28,38,49, \\ldots\n$$\n\nIs it possible that 793210041 belongs to this sequence?",
        "answer": "793210041",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74283,
        "difficulty": null
      },
      {
        "id": "0cfe13046b3d7e8b7e2a87cbd2626b0662d2e2291e25209f62924deb5164273d",
        "domain": "numina",
        "question": "The sequence $a_{1}, a_{2}, a_{3}, \\ldots$ is an arithmetic sequence with common difference 3 and $a_{1}=1$. The sequence $b_{1}, b_{2}, b_{3}, \\ldots$ is an arithmetic sequence with common difference 10 and $b_{1}=2$. What is the smallest integer larger than 2023 that appears in both sequences?",
        "answer": "2032",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194473,
        "difficulty": null
      },
      {
        "id": "222fa7545237dedc787ba0ba7b14006886d53e50b80a4b4fbd9d79da63dffba7",
        "domain": "numina",
        "question": "5. The edges of the cuboid are in the ratio $4: 3$, the diagonals of the lateral faces are in the ratio $\\sqrt{20}: \\sqrt{13}$, and the area of the diagonal section is in the ratio 2:1 to the volume of the cuboid. Determine the area and volume of this cuboid.",
        "answer": "\\frac{125}{576}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188193,
        "difficulty": null
      },
      {
        "id": "c508086a24eb8a9227748ca53e9aefe86f8982d09e158e017438d5f987ec46fc",
        "domain": "numina",
        "question": "Determine a point $N$ on the directrix $FM$ of a parabola or its extension, such that the distances from $N$ to the focus and the directrix differ by a given length, $l$.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 175881,
        "difficulty": null
      },
      {
        "id": "b4130dc6a3c0189df791a3303f37371cdf7fdee0329c7941cfd2d93d34abf0e4",
        "domain": "numina",
        "question": "4. On the island, there live knights who always tell the truth and liars who always lie. In the island's football team, there are 11 people. Player number 1 said: \"In our team, there are as many knights as there are liars.\" Player number 2 said: \"In our team, the number of knights and the number of liars differ by one,\" and so on. Player number 11 said: \"In our team, the number of knights and the number of liars differ by ten.\" How many knights are in the team, and if there are any, what are their numbers? Find all possible answers to this question.",
        "answer": "0",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12611,
        "difficulty": null
      },
      {
        "id": "17311723157639c7a56ec1d335ffc3576fd3e42fc0a03b1ac779466eefe764fe",
        "domain": "numina",
        "question": "35. What is the smallest positive integer $n$, where $n \\neq 11$, such that the highest common factor of $n-11$ and $3 n+20$ is greater than 1 ?",
        "answer": "64",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160904,
        "difficulty": null
      },
      {
        "id": "b9416464ae8659a5bc991d6199454212dbb23e6a968b5effe4804d006a931ab1",
        "domain": "numina",
        "question": "Let's determine the remainder of the sum $1+22+333+4444+\\ldots+\\underbrace{s s s . \\ldots s}_{s \\text { times }}$ when divided by $(s-1)$ in the base $s+1$ number system.\n\n---\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\nDetermine the remainder of the sum $1+22+333+4444+\\ldots+\\underbrace{s s s . \\ldots s}_{s \\text { times }}$ when divided by $(s-1)$ in the base $s+1$ number system.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 170248,
        "difficulty": null
      },
      {
        "id": "262be5701fa6dbdf99d0b3060006f7705ee08ee57a18fb11d2e6157a00c4ecd9",
        "domain": "numina",
        "question": "216. Blindness in Bats. One naturalist, trying to mystify Colonel Crackham, told him that he had studied the question of blindness in bats.\n\n- I found,- he said, - that the ingrained habit of bats to sleep during the day in dark corners and fly only at night has led to the spread of blindness among them, although some individuals saw well with both or one eye. Two of the bats I examined saw with the right eye, three with the left, four did not see with the left eye, and five did not see with the right eye.\n\nCould you calculate the smallest number of bats the naturalist had to examine to obtain such results?",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 41132,
        "difficulty": null
      },
      {
        "id": "e3519964cc371f982c72fd963470dc4986eb2cae0a9c2e916f2d02cae3e56a8d",
        "domain": "numina",
        "question": "4. (5 points) Count the number of squares in the figure: $\\qquad$.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "18",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 143365,
        "difficulty": null
      },
      {
        "id": "7b8a14210bb677c14129c6b45784b7e9b464767a089c52674d3018dfb62e9ba0",
        "domain": "numina",
        "question": "$\\underline{\\text { Folklore }}$\n\nGiven a polynomial $P(x)$ with integer coefficients. It is known that $P(1)=2013, P(2013)=1, P(k)=k$, where $k$ is some integer. Find $k$.\n\n#",
        "answer": "1007",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 67970,
        "difficulty": null
      },
      {
        "id": "2db1517e87d24d5a14f0e8de28a6ac03e84cddcd3a42c496359e85cac69afea1",
        "domain": "numina",
        "question": "3.13. $\\left\\{\\begin{array}{l}\\left(x_{3}+x_{4}+x_{5}\\right)^{5}=3 x_{1}, \\\\ \\left(x_{4}+x_{5}+x_{1}\\right)^{5}=3 x_{2}, \\\\ \\left(x_{5}+x_{1}+x_{2}\\right)^{5}=3 x_{3}, \\\\ \\left(x_{1}+x_{2}+x_{3}\\right)^{5}=3 x_{4}, \\\\ \\left(x_{2}+x_{3}+x_{4}\\right)^{5}=3 x_{5} .\\end{array}\\right.$\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_3d5ef740640a71a48194g-033.jpg?height=402&width=443&top_left_y=787&top_left_x=1155)\n\n## 3.3. Positive solutions",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 38154,
        "difficulty": null
      },
      {
        "id": "3e2df036ef0ed122f759a8b14b5d931531c42200df46252f795be9e700a0a885",
        "domain": "numina",
        "question": "$10 x$ is a positive integer, the last two digits of $3^{x}$ form a prime number, then the two values of $x$ closest to 2008 are $\\qquad$ and $\\qquad$ .",
        "answer": "20082009",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 125565,
        "difficulty": null
      },
      {
        "id": "e0076d218c9e5b60c036d24c0a3b21e54ac97a6d12c2efc5f5a266b7f9043130",
        "domain": "numina",
        "question": "11. As shown in the figure, in the Cartesian coordinate system, the ellipse $\\Gamma$ : $\\frac{x^{2}}{2}+y^{2}=1$ has left and right foci $F_{1}, F_{2}$, respectively. Let $P$ be a point on $\\Gamma$ in the first quadrant, and the extensions of $P F_{1}, P F_{2}$ intersect $\\Gamma$ at points $Q_{1}\\left(x_{1}, y_{1}\\right), Q_{2}\\left(x_{2}, y_{2}\\right)$, respectively. Find the maximum value of $y_{1}-y_{2}$.",
        "answer": "\\frac{2\\sqrt{2}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 104899,
        "difficulty": null
      },
      {
        "id": "035e081d21fddfa7f8e483a2a6a041ede23a6111468b01ec9a4ae12c68ae31a1",
        "domain": "numina",
        "question": "The squares of the natural numbers from 1 to 99 were written one after another, forming the number 14916253649... What is the digit that occupies the 100th position? (The positions are counted from left to right, so the $1^{\\underline{a}}$ position is the 1, the $2^{\\underline{a}}$ is the 4, and so on.)",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74920,
        "difficulty": null
      },
      {
        "id": "daccff0450907691b24c005e69490390df2a9c8546728cf36925b7559a5975f5",
        "domain": "numina",
        "question": "A lead bullet at $20^{\\circ}$ temperature, completely inelastic, falls freely from a height of $100 \\mathrm{~m}$ onto a solid plane. If all the apparently lost energy was converted into heat absorbed by the bullet, the questions are:\n\n1. What is the temperature of the bullet immediately after the fall?\n2. What speed would the bullet need to be dropped from above to raise its temperature to the melting point of lead?\n\nThe specific heat of lead $C=0.0315$, the mechanical equivalent of heat $E=425$, the melting point of lead $T=335^{\\circ}$, and the intensity of gravitational force $g=9.80 \\frac{\\mathrm{m}}{\\mathrm{s}^{2}}$.",
        "answer": "93.2",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 171890,
        "difficulty": null
      },
      {
        "id": "5a3dd69cda4f784975ddf5d51cdbd7bec9637bd33779c0c541aa054bc881e9b1",
        "domain": "numina",
        "question": "Problem 4. Determine the four-digit number $\\overline{x y z t}$ for which\n\n$$\n\\overline{x y z t}+4 \\cdot \\overline{y z t}+2 \\cdot \\overline{z t}=2018\n$$",
        "answer": "1174",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 189031,
        "difficulty": null
      },
      {
        "id": "288103217b799ec8e73a08267f2ee78aecf760810a1854c97dc78a97d61a014f",
        "domain": "numina",
        "question": "42nd Swedish 2002 Problem 2 A, B, C can walk at 5km/hr. They have a car that can accomodate any two of them whch travels at 50km/hr. Can they reach a point 62km away in less than 3 hrs?",
        "answer": "2.95",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 150815,
        "difficulty": null
      },
      {
        "id": "2cf314d87145419ca02654bc74e3a3d96b4520fccd3d6db36e69763c3739ffce",
        "domain": "numina",
        "question": "In the zoo, there were 200 parrots. One day, they each made a statement in turn. Starting from the second,\n\nthe statements were \"Among the previously made statements, more than 70% are false.\" How many false statements did the parrots make\n\n#",
        "answer": "140",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56060,
        "difficulty": null
      },
      {
        "id": "ee72d531fb0100e13b23ab688ff4fc4900dc8a2945c7d91275397d557eaabb7b",
        "domain": "numina",
        "question": "Problem 2. To number the pages of a textbook, 411 digits are needed. How many pages does the textbook have?",
        "answer": "173",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188534,
        "difficulty": null
      },
      {
        "id": "05d38d14f448127d482fda600a3fde2d28beb4df877aee8b10a6e1c4c36d75d5",
        "domain": "numina",
        "question": "2. the real numbers $a_{1}, a_{2}, \\ldots, a_{16}$ fulfill the two conditions\n\n$$\n\\sum_{i=1}^{16} a_{i}=100 \\quad \\text { and } \\quad \\sum_{i=1}^{16} a_{i}^{2}=1000\n$$\n\nWhat is the largest possible value that $a_{16}$ can take?\n\n## Solution",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91217,
        "difficulty": null
      },
      {
        "id": "af983c461dfe96b1853a7253a7c0ada180002269b1b008a5e6617277d2757ea1",
        "domain": "numina",
        "question": "5. The numbers from 100 to 999 are written without spaces. What is the remainder when the resulting 2700-digit number is divided by 7?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 10293,
        "difficulty": null
      },
      {
        "id": "f99a4f4f2d02945de021756078375bad9b2cd209941a5d7eae19338b066ca8d9",
        "domain": "numina",
        "question": "\\begin{array}{l}\n\\text { [The ratio in which the bisector divides the side. } \\\\ \\text { [The ratio of the areas of triangles with a common base or common height] }]\\end{array}$\n\nIn triangle $ABC$, sides $AB$ and $BC$ are equal to each other, $AC=2$, and $\\angle ACB=30^{\\circ}$. From vertex $A$ to the side $BC$, the bisector $AE$ and the median $AD$ are drawn. Find the area of triangle $ADE$.",
        "answer": "\\frac{2\\sqrt{3}-3}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59854,
        "difficulty": null
      },
      {
        "id": "5d8deaac016fd181e5b4fb957702531d3b8a6021216464b55d8859e201d1d9a2",
        "domain": "numina",
        "question": "6.5. On the board, there are 5 integers. By adding them in pairs, the following set of 10 numbers was obtained: $-1,5,8,9,11,12,14,18,20,24$. Determine which numbers are written on the board. In your answer, write their product.",
        "answer": "-2002",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6416,
        "difficulty": null
      },
      {
        "id": "1f42b579210ec126d2da8e65cbb3acf123165d035753a99995f0e3fa2cb4dd41",
        "domain": "numina",
        "question": "## Task A-2.5.\n\nDetermine all pairs $\\{a, b\\}$ of distinct real numbers such that the equations\n\n$$\nx^{2} + a x + b = 0 \\quad \\text{and} \\quad x^{2} + b x + a = 0\n$$\n\nhave at least one common solution in the set of real numbers.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89083,
        "difficulty": null
      },
      {
        "id": "95d36546f7901c03a8cffc8c547280c73859a4d289fc670f94b3ea5e7d2211a1",
        "domain": "numina",
        "question": "11. Given $z_{1}, z_{2}, z_{3} \\in \\mathbb{C}$ and $\\left.\\mid z_{1}\\right\\}=\\left\\{z_{2} \\mid=\\sqrt{5}\\right\\} z_{3} \\mid$ and $z_{1}+z_{3}=z_{2}$, then $\\frac{z_{1} z_{2}}{z_{3}^{2}}=$",
        "answer": "-5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110227,
        "difficulty": null
      },
      {
        "id": "e5578a5abfeea8c4595e396955aa8973120e2bfac1a5794d1c677b68afc07c7b",
        "domain": "numina",
        "question": "Draw two circles with the same center, one with a radius of $1 \\mathrm{~cm}$ and the other with a radius of $3 \\mathrm{~cm}$. In the region outside the circle with a radius of $1 \\mathrm{~cm}$ and inside the circle with a radius of $3 \\mathrm{~cm}$, draw circles that are simultaneously tangent to both circles, as shown in the given figure.\n(a) What should be the radius of these circles?\n\n(b) What is the maximum number of these circles that can be drawn without overlapping?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_280b19a87fa67143351eg-031.jpg?height=252&width=257&top_left_y=248&top_left_x=1590)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74472,
        "difficulty": null
      },
      {
        "id": "d17511eb9e20b465c33a181daf00803abc394c74e03b04c24afc79f7f239d87f",
        "domain": "numina",
        "question": "2. On a sheet of lined paper, two rectangles are outlined. The first rectangle has a vertical side shorter than the horizontal side, while the second has the opposite. Find the maximum possible area of their common part, if the first rectangle contains 2015 cells, and the second - 2016.",
        "answer": "1302",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11841,
        "difficulty": null
      },
      {
        "id": "614ddf0f26344c008d3183226fbaea1a954074f7d67a976e1d5f7b51ab5af239",
        "domain": "numina",
        "question": "5.082 If we expand all the brackets in the expression $(1+x)^{9}+(1+x)^{10}+\\ldots$ $\\ldots+(1+x)^{14}$ and combine like terms, we will get some polynomial. Determine the coefficient of $x^{9}$ in this polynomial without expanding the brackets.",
        "answer": "3003",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 52301,
        "difficulty": null
      },
      {
        "id": "1666d30868fc820b9647c0e9c5df99ef2a6870be6c7be62ee518c18a3bdc9c10",
        "domain": "numina",
        "question": "20. [12] Let $f(x)=x^{3}-3 x$. Compute the number of positive divisors of\n$$\n\\left\\lfloor f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(\\frac{5}{2}\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right\\rfloor\n$$\nwhere $f$ is applied 8 times.",
        "answer": "6562",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151885,
        "difficulty": null
      },
      {
        "id": "9459d63e0ae8f915e11b84eb62e9123eddeb046e450560422540b3bb96fe8076",
        "domain": "numina",
        "question": "## Task A-1.4.\n\nIn the plane, one hundred circles with the same center and radii $1,2, \\ldots, 100$ are drawn. The smallest circle is colored red, and each of the 99 circular rings bounded by two circles is colored red or green such that adjacent areas are of different colors.\n\nDetermine the total area of the green-colored regions.",
        "answer": "5050\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88912,
        "difficulty": null
      },
      {
        "id": "bf22d239f7aa70cbcf0d34e5c64b428cbd3900d0bc0c377ea8a231d70189fa8c",
        "domain": "numina",
        "question": "Shapovalov A.v.\n\nThere are 100 numbers. When each of them was increased by 1, the sum of their squares did not change. Each number was increased by 1 again.\n\nWill the sum of the squares change this time, and if so, by how much?",
        "answer": "200",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 53134,
        "difficulty": null
      },
      {
        "id": "fd5361c7163eceebba83c6d3be935c56612cabe51467d1a4e1422448795afa24",
        "domain": "numina",
        "question": "Subject 2\n\nSolve in $\\mathbb{R}$ the equation: $\\frac{1}{\\{x\\}}=\\frac{1}{x}+\\frac{1}{[x]}$, where $\\{x\\}$ represents the fractional part of $x$ and $[x]$ represents the integer part of $x$.",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178995,
        "difficulty": null
      },
      {
        "id": "5da0c95ee7d43ae391ceec6eecb0cd035077483df069d614926e7d8fd49706e4",
        "domain": "numina",
        "question": "3. Given the sequence $\\left\\{x_{n}\\right\\}$, and $x_{n+1}=\\frac{x_{n}+(2-\\sqrt{3})}{1-x_{n}(2-\\sqrt{3})}$, then $x_{1001}-x_{401}=$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134915,
        "difficulty": null
      },
      {
        "id": "7453a5e21511fb1ccd1160c2e6d277f18f66ec34a2c07f6e2b6b97819fd02ed4",
        "domain": "numina",
        "question": "Three. (50 points) Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $a_{1}=1599$, when $k^{2}<n<(k+1)^{2}$, $a_{n}=\\left[\\frac{k+1}{2}\\right]+\\left[\\frac{k}{2}\\right]+1\\left(k \\in \\mathbf{N}^{*}\\right)$, does there exist a positive integer $n$ such that the equation $\\sum_{i=1}^{n} a_{i}=2006$ is satisfied? If it exists, find all such numbers $n$; if not, explain the reason.",
        "answer": "68",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 108836,
        "difficulty": null
      },
      {
        "id": "3f53cf3b44b1ae6c797dfb2870bd12b5f7bd023aa535e92fd6550930d8703655",
        "domain": "numina",
        "question": "If in a right-angled triangle we draw the midlines $K_{1}$ and $K_{2}$ corresponding to the legs $a$ and $b$, then\n\n$$\n16 K_{2}^{2}-4 K_{1}^{2}=15 a^{2}\n$$",
        "answer": "15^{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173296,
        "difficulty": null
      },
      {
        "id": "c97c9406a610c09a176de3c912a8ddd2a5592681fb0891374c193e24ef72af17",
        "domain": "numina",
        "question": "$$\n\\begin{array}{l}\n\\text { 10. Given } f(x)=\\left\\{\\begin{array}{ll}\n-2 x, & x<0 ; \\\\\nx^{2}-1, & x \\geqslant 0,\n\\end{array}\\right. \\text { the equation } \\\\\nf(x)+2 \\sqrt{1-x^{2}}+\\left|f(x)-2 \\sqrt{1-x^{2}}\\right|-2 a x-4=0\n\\end{array}\n$$\n\nhas three real roots $x_{1}<x_{2}<x_{3}$. If $x_{3}-x_{2}=2\\left(x_{2}-x_{1}\\right)$, then the real number $a=$ $\\qquad$",
        "answer": "\\frac{\\sqrt{17}-3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124645,
        "difficulty": null
      },
      {
        "id": "659db059af2e3b464777973d044b78b6ae6ffa3e8c3417a5e1b573b05c205dcb",
        "domain": "numina",
        "question": "Example 2.5. $I=\\int_{-1}^{1} x|x| d x$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\nHowever, since the text provided is already in a form that is commonly used in English for mathematical expressions, the translation is essentially the same:\n\nExample 2.5. $I=\\int_{-1}^{1} x|x| d x$.",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30847,
        "difficulty": null
      },
      {
        "id": "9231f03e83f4862c788d73378bc97a72420e811968d28336de4cec887031089e",
        "domain": "numina",
        "question": "In a group of 200 people, only $1 \\%$ are women. Determine the number of men who must leave the group so that $98 \\%$ of the remaining people are male.\n\n#",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73936,
        "difficulty": null
      },
      {
        "id": "ed48ef160d0903de89da8dc5b98d67035489abf31f58dc21d66a60a05147aece",
        "domain": "numina",
        "question": "88. The edge of a regular tetrahedron is $a$. The plane $\\boldsymbol{P}$ passes through the vertex $B$ and the midpoints of the edges $A C$ and $A D$. A sphere touches the lines $A B, A C, A D$ and that part of the plane $P$ which is enclosed within the tetrahedron. Find the radius of the sphere.",
        "answer": "\\frac{\\sqrt{2}}{5\\\\sqrt{11}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29797,
        "difficulty": null
      },
      {
        "id": "af85e2d5a77193ea0ae1da44bb418dd8cde718434169ad868bd86a16cff57ebf",
        "domain": "numina",
        "question": "[ Divisibility of numbers. General properties ]  $[$ Classical combinatorics (other). $]$\n\nHow many natural numbers less than a thousand are not divisible by either 5 or 7?",
        "answer": "686",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68590,
        "difficulty": null
      },
      {
        "id": "609ae07acb80fa94d54f57c23996c903f71aef16a7160451856014ae1c9e969c",
        "domain": "numina",
        "question": "18. Collinear points $A, B$, and $C$ are given in the Cartesian plane such that $A=(a, 0)$ lies along the $x$-axis, $B$ lies along the line $y=x, C$ lies along the line $y=2 x$, and $A B / B C=2$. If $D=(a, a)$, the circumcircle of triangle $A D C$ intersects $y=x$ again at $E$, and ray $A E$ intersects $y=2 x$ at $F$, evaluate $A E / E F$.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151296,
        "difficulty": null
      },
      {
        "id": "38501eb1503316faeb176a24ed3dd03e2d607d62815f58a447d3a96ded7deadb",
        "domain": "numina",
        "question": "Example 14 Find the number of pairwise non-congruent triangles with vertices at the vertices of a regular 2000-gon.\n\n保留源文本的换行和格式，翻译结果如下：\n\nExample 14 Find the number of pairwise non-congruent triangles with vertices at the vertices of a regular 2000-gon.",
        "answer": "333333",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144568,
        "difficulty": null
      },
      {
        "id": "0a971b2b5d11d43d767fcb27288c786836debd72d857b78ced0d349e2644d3f6",
        "domain": "numina",
        "question": "In a fruit shop, Jaime noticed that an orange costs the same as half an apple plus half a real, and he also noticed that a third of an apple costs the same as a quarter of an orange plus half a real. With the value of 5 oranges plus 5 reals, how many apples can Jaime buy?\n\n#",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73599,
        "difficulty": null
      },
      {
        "id": "05a1bb0a4a21b08c160e465ab9f11ed1816c9ed4767db5a46f989f188a1bd2a6",
        "domain": "numina",
        "question": "8. (1 mark) Given that $0.3010<\\log 2<0.3011$ and $0.4771<\\log 3<0.4772$. Find the leftmost digit of $12^{37}$.\n(1 分) 設 $0.3010<\\log 2<0.3011$ 及 $0.4771<\\log 3<0.4772$, 求 $12^{37}$ 最左的一位數字。",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153453,
        "difficulty": null
      },
      {
        "id": "1e163b895d595333eeebbcb1f83740a503441769c6dfa0e781188d1948164b2f",
        "domain": "numina",
        "question": "4. If the orthocenter of $\\triangle O A B$ is exactly the focus of the parabola $y^{2}=4 x$, where $O$ is the origin, $A, B$ are on the parabola, then the area $S$ of $\\triangle O A B$ is $\\qquad$ .",
        "answer": "10\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 117061,
        "difficulty": null
      },
      {
        "id": "e2cba495b1277e4ecee81af5bc3cdc54185ac8eeb6b808b7057326ff2f5c73e5",
        "domain": "numina",
        "question": "I1.4 If $\\log _{2} a+\\log _{2} b \\geq \\gamma$, determine the smallest positive value $\\delta$ for $a+b$.",
        "answer": "16",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 153375,
        "difficulty": null
      },
      {
        "id": "de815fabfbea0db489a3fb49f4552a9ecd2e72a131067049af02ce892023273c",
        "domain": "numina",
        "question": "Example 4 (2004 American Invitational Mathematics Examination) A circle of radius 1 is randomly placed inside a $15 \\times 36$ rectangle $ABCD$. Find the probability that this circle does not intersect with the diagonal $AC$.",
        "answer": "\\frac{375}{442}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 109785,
        "difficulty": null
      },
      {
        "id": "fbeaa848e02938473ef8c66d159365fe22789ba412c9e96c66c7fdfc8ed193fb",
        "domain": "numina",
        "question": "4. Let $a \\triangle b$ and $a \\nabla b$ represent the minimum and maximum of the two numbers $a$ and $b$, respectively. For example, $3 \\triangle 4=3, 3 \\nabla 4=4$. For different natural numbers $x$, the number of possible values for $6 \\triangle[4 \\nabla(x \\triangle 5)]$ is $\\qquad$.",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 100607,
        "difficulty": null
      },
      {
        "id": "5c652f5a8d945412a69af70e17b705d3bc1097d6cae5c0676a702e090b0e5bee",
        "domain": "numina",
        "question": "While investigating a case, Detective Bashkovsky discovered that the key witness was the one from the Petrovs family who came home before the others on that fateful day. The investigation revealed the following facts.\n\n1. Neighbor Marya Kuzminichna wanted to borrow some salt from the Petrovs, rang their doorbell, but no one answered. When? Who knows? It was already dark...\n2. Galina Efimovna Petrova, coming home in the evening, found both children in the kitchen, and her husband on the sofa - he had a headache.\n3. Husband Anatoly Ivanovich claimed that as soon as he came home, he immediately lay down on the sofa and dozed off, saw no one, heard nothing, the neighbor definitely didn't come - the doorbell would have woken him up.\n4. Daughter Svetlana said that upon returning home, she immediately went to her room, knew nothing about her father, but as usual, stumbled over Dima's shoe in the hallway.\n5. Dmitry didn't remember when he came home, didn't see his father, but heard Svetka swearing about the shoe.\n\n\"Hmm,\" thought Bashkovsky. \"What is the probability that Dmitry returned home before his father?\"",
        "answer": "\\frac{2}{3}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 55952,
        "difficulty": null
      },
      {
        "id": "b065e2a890e2037c188bef5232d55790d8d80f03868f48d5681084edae1d876e",
        "domain": "numina",
        "question": "11.4. (SFRY, 70). The diagonals of a convex pentagon $A B C D E$, intersecting, form a pentagon $A_{1} B_{1} C_{1} D_{1} E_{1}$ and a five-pointed star.\n\na) Find the sum of the angles of this star at the vertices $A$, $B, C, D, E$.\n\nb) Find the ratio of the area of the pentagon $A_{1} B_{1} C_{1} D_{1} E_{1}$ to the area of the pentagon $A B C D E$ under the condition of the regularity of the latter.",
        "answer": "\\frac{7-3\\sqrt{5}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 28990,
        "difficulty": null
      },
      {
        "id": "232b742336759681fa19876539789b92468a1a2af41a1fc3a7a8bcd5e46a61d2",
        "domain": "numina",
        "question": "3. In a square $A B C D$ with side length 2, $E$ is the midpoint of $A B$. Now, fold the figure along segments $E C, E D$ so that segments $E A, E B$ coincide. This results in a tetrahedron $C D E A$ (point $B$ has coincided with point $A$). The volume of this tetrahedron is",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137324,
        "difficulty": null
      },
      {
        "id": "25ddef0d3d2782cff320d36ab36a96a5b380b78fcd8070e7baf151bf6dceab87",
        "domain": "numina",
        "question": "Example 7 (2006 Beijing College Entrance Examination Final Question) In the sequence $\\left\\{a_{n}\\right\\}$, if $a_{1}, a_{2}$ are positive integers, and $a_{n}=\\left|a_{n-1}-a_{n-2}\\right|$, $n=3,4,5, \\cdots$, then $\\left\\{a_{n}\\right\\}$ is called an \"absolute difference sequence\".\n(1) Give an example of an \"absolute difference sequence\" whose first five terms are not zero (only the first ten terms are required to be written);\n(2) If in the \"absolute difference sequence\" $\\left\\{a_{n}\\right\\}$, $a_{20}=3, a_{21}=$ 0, and the sequence $\\left\\{b_{n}\\right\\}$ satisfies $b_{n}=a_{n}+a_{n+1}+a_{n+2}, n=1,2$, $3, \\cdots$, determine whether the limits of $a_{n}$ and $b_{n}$ exist as $n \\rightarrow \\infty$. If they exist, find their limit values;\n(3) Prove: Any \"absolute difference sequence\" contains infinitely many zero terms.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133444,
        "difficulty": null
      },
      {
        "id": "630b5e2a0a3f8b68b72cbce1f44364daecfccfcc24a84743438628b5edea71a6",
        "domain": "numina",
        "question": "# Task No. 7.4\n\n## Condition:\n\nProfessor Severus Snape has prepared three potions, each with a volume of 480 ml. The first potion makes the person who drinks it smart, the second makes them beautiful, and the third makes them strong. To ensure the potion works, one needs to drink at least 30 ml of it. Severus Snape was about to drink his potions when he was called to the headmaster's office and left, leaving the labeled potions in large jugs on his desk. Taking advantage of his absence, Harry, Hermione, and Ron approached the table with the potions and began to sample them.\n\nHermione was the first to try the potions: she approached the first jug with the potion of wisdom and drank half of it, then poured the remainder into the second jug with the potion of beauty, thoroughly mixed the contents, and drank half of it. Next, it was Harry's turn: he drank half of the third jug with the potion of strength, then poured the remainder into the second jug, thoroughly mixed everything in this jug, and drank half of it. Now all the contents were in the second jug, which was left for Ron. What percentage of the contents of this jug does Ron need to drink to ensure that each of the three potions will definitely affect him?",
        "answer": "50",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 15633,
        "difficulty": null
      },
      {
        "id": "933922f52b1f7eedbb1da022484dd5fb47c93ab6f094a7f4b18790a67881d085",
        "domain": "numina",
        "question": "1. (2 points) For the polynomial $\\left(x^{2}-x+1\\right)^{100}$, find the sum of the coefficients of the even powers of $x$.",
        "answer": "\\frac{1+3^{100}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8803,
        "difficulty": null
      },
      {
        "id": "957deb6a67090ad2c33cc1db0c0132c37d4829224e0af6829f0bf8e63fbec4d8",
        "domain": "numina",
        "question": "Problem 4. A row of 11 numbers is written such that the sum of any three consecutive numbers is 18. Additionally, the sum of all the numbers is 64. Find the central number.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 18326,
        "difficulty": null
      },
      {
        "id": "04503d63a1a0b0a576f13f980a2115a0890ec1f822b81362fb81bc9cbb2b735e",
        "domain": "numina",
        "question": "Example 6. A continuous random variable $X$ is given by the probability density function $p(x)=\\cos x$ in the interval $(0 ; \\pi / 2)$; outside this interval, $p(x)=0$. Find the expected value of the function $Y=X^{2}$.",
        "answer": "\\frac{\\pi^{2}}{4}-2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33040,
        "difficulty": null
      },
      {
        "id": "2c9bdde9e8efd172341e2f14ca5dd4f56c33d5954453b595449ca740fb38d6fe",
        "domain": "numina",
        "question": "A fishing cooperative wanted to get an estimate of how many specimens of a certain fish species, which have already reached a certain size, live in one of their ponds. (Hereafter, these will be referred to as suitable fish.) For this purpose, they marked 50 suitable fish from their daily catch and released them back into the water. The next day, in their catch from 3 different locations in the pond, they counted a total of 67, 94, and 43 suitable fish, respectively, and among these, 6, 10, and 3 were marked. How could they estimate the number of suitable fish after the second day's catch?",
        "answer": "537",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 167599,
        "difficulty": null
      },
      {
        "id": "bf09219b45ae8d87d74a8312ee4e02586d8577559b6cb356ad1a74d91df9907f",
        "domain": "numina",
        "question": "84 (1194). a) Find the sum of the cubes of the roots of the equation $2 x^{2}-$ $-5 x+1=0$.\n\nb) Find the value of the expression $\\frac{x_{1}}{x_{2}}+\\frac{x_{2}}{x_{1}}$, where $x_{1}$ and $x_{2}-$ are the roots of the equation $2 x^{2}-11 x+13=0$.",
        "answer": "2\\frac{17}{26}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36113,
        "difficulty": null
      },
      {
        "id": "c35e340bc8268d3d41b49b1158b99b6f38ce822fcda7abf60b377f7cf85d0621",
        "domain": "numina",
        "question": "\\section*{Problem 15}\n\nWhat is the minimal value of \\(\\mathrm{b} /(\\mathrm{c}+\\mathrm{d})+\\mathrm{c} /(\\mathrm{a}+\\mathrm{b})\\) for positive real numbers \\(\\mathrm{b}\\) and \\(\\mathrm{c}\\) and nonnegative real numbers a and \\(\\mathrm{d}\\) such that \\(\\mathrm{b}+\\mathrm{c} \\geq \\mathrm{a}+\\mathrm{d}\\) ?",
        "answer": "\\sqrt{2}-\\frac{1}{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 187034,
        "difficulty": null
      },
      {
        "id": "4afd1f680bcb8eb0b6e337e8224c9f055a56daef2e0618675e9004e0e88607fc",
        "domain": "numina",
        "question": "2. Given $-\\frac{\\pi}{2}<\\alpha<\\frac{\\pi}{2}, 2 \\tan \\beta=\\tan 2 \\alpha, \\tan (\\beta-\\alpha)=-2 \\sqrt{2}$. Then $\\cos \\alpha=$",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115402,
        "difficulty": null
      },
      {
        "id": "07d07144a69874088346fd4f18416905ed32ea0491c342ebcbe1bc52ddc9a1f1",
        "domain": "numina",
        "question": "11. A circle is drawn inside a regular hexagon so that it touches all six sides of the hexagon. The area of the circle is $\\pi \\times 64 \\sqrt{3}$. What is the area of the hexagon?",
        "answer": "384",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156381,
        "difficulty": null
      },
      {
        "id": "00d38f059c549991db9b5c541290aff666e9d1a198b6a48afd6e13dc5bd7d770",
        "domain": "numina",
        "question": "# 2. Inflation over two years will be\n\n$$\n\\left((1+0.025)^{\\wedge 2-1}\\right)^{*} 100 \\% = 5.0625 \\%\n$$\n\nThe real return on a bank deposit, taking into account the extension for the second year, will be $(1.06 * 1.06 / (1+0.050625) - 1) * 100 = 6.95 \\%$\n\n## Evaluation Criteria:\n\nA maximum of 25 points for a correct and justified solution. Of these:\n\n2 points for the correct calculation of the investment amount.\n\n5 points for the correct calculation of the nominal return on the bank deposit. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- 3 points for calculating the amount the family can withdraw after closing the deposit, taking into account its extension for the second year.\n- 1 point for calculating the amount the family can withdraw after closing the deposit, without correctly accounting for its extension for the second year.\n\n7 points for the correct calculation of the expected return on investment in construction. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- a maximum of 2 points for conceptually incorrect consideration of the probabilities of different outcomes.\n- 2 points for correctly accounting for the 13% income tax.\n\n1 point for the correct choice of investment method. If the correct choice was made based on incorrect calculations of the return in any of the investment options, the participant did not receive a point for this part of the solution.\n\n10 points for the correct calculation of the return. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- 4 points for a logically correct method of calculating the return in %, provided that the real return was calculated incorrectly.\n- 8 points for calculating the return in % as the difference between the nominal return and inflation over 2 years, provided that the nominal return was calculated correctly.\n- 1 point for incorrect accounting of inflation over two years.\n\nIf the entire logic of the calculations is followed, but the participant makes only arithmetic errors, then -1 point for each such error.\n\n## Task 5\n\nThe banking sector has always been a tempting target for criminals, so commercial and government banks are implementing more and more protection technologies.\n\nYou head the information security department of a large commercial bank. Propose your protection strategy for the following objects of the banking infrastructure (at least one existing method for each object):\n\n- employee profiles;\n- bank cards;\n- ATMs;\n- online banking;\n- online store payments;\n- central automated banking system.",
        "answer": "6.95",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11442,
        "difficulty": null
      },
      {
        "id": "ffd4c730f5b59a7fb0e03f82e4d6be2cb8b1cefacecd1a22d7784f41842caa03",
        "domain": "numina",
        "question": "4. If the functions $f(x)$ and $g(x)$ are defined on $\\mathbf{R}$, and $f(x-y)=f(x) g(y)-g(x) f(y)$, if $f(-2)=f(1) \\neq 0$, then $g(1)+g(-1)=$ (answer with a number).\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "-1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 106127,
        "difficulty": null
      },
      {
        "id": "db8e3e9abadb2486c7f69165f58efcbb2dbee6da4fb66e1cc387b7434ffe5041",
        "domain": "numina",
        "question": "34. For an earth excavation project, Team A can complete it alone in 16 days, and Team B can complete it alone in 20 days. Now, both teams are working together, and their work efficiency increases by $20 \\%$. When $\\frac{1}{4}$ of the project is completed, they suddenly encounter groundwater, which affects the construction progress, reducing the daily excavation by 47.25 cubic meters of soil. As a result, the entire project took 10 days to complete. The total amount of soil to be excavated for the entire project is $\\qquad$ cubic meters.",
        "answer": "1100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105416,
        "difficulty": null
      },
      {
        "id": "0580452d5145cc7f6f6967a46ad1baa8db98e9d862342bdb04f6a8b4dfbaf019",
        "domain": "numina",
        "question": "16. On the island of Friends and Foes, every citizen is either a Friend (who always tells the truth) or a Foe (who always lies). Seven citizens are sitting in a circle. Each declares \"I am sitting between two Foes\". How many Friends are there in the circle?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 156346,
        "difficulty": null
      },
      {
        "id": "4758d37f0e90a010eedf5cbf1f712c6bfdb1d02c077e56ad3832f1c6f95450b2",
        "domain": "numina",
        "question": "13. 3. 13 ${ }^{\\star \\star}$ The axial cross-section of a large wine glass is the graph of the function $y=x^{4}$. A small ball with radius $r$ is placed into the glass. When the radius $r$ is maximized, the ball can touch the lowest point of the bottom of the glass (i.e., what is the maximum value of $r$ such that a circle with radius $r$ located in the region $y \\geqslant x^{4}$ can pass through the origin)?",
        "answer": "\\frac{3}{2}\\sqrt[3]{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127007,
        "difficulty": null
      },
      {
        "id": "77bbd312d34a3f39a91d5e0a5e76d0e1f963c2d1293b7d4eb63f041d300e9da5",
        "domain": "numina",
        "question": "For $B_{n}\\left(n \\in \\mathbf{N}^{*}\\right)$. Let the point with coordinates $(1,1)$ be $M, P_{n}\\left(x_{n}, y_{n}\\right)$ is the circumcenter of $\\triangle A_{n} B_{n} M$, find the value of $x_{1}+x_{2}+\\cdots+x_{100}$.",
        "answer": "200\\frac{50}{101}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142140,
        "difficulty": null
      },
      {
        "id": "95549ed8ff575c41ce6bfb1f5c7a23de76153e153eee372ba7141b8a7faaed59",
        "domain": "numina",
        "question": "The sum of non-negative numbers $x_{1}, x_{2}, \\ldots, x_{10}$ is 1. Find the maximum possible value of the sum $x_{1} x_{2}+$ $x_{2} x_{3}+\\ldots+x_{9} x_{10}$",
        "answer": "0.25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70204,
        "difficulty": null
      },
      {
        "id": "ffc09b93ab9fa3885b383650d4849fd30e0c147a6a02260efa98628528f1cc2e",
        "domain": "numina",
        "question": "34th CanMO 2002 Problem 1 What is the largest possible number of elements in a subset of {1, 2, 3, ... , 9} such that the sum of every pair (of distinct elements) in the subset is different?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 148687,
        "difficulty": null
      },
      {
        "id": "05117efc1e4d5e796afd13b2a37c8315db4204b53a521816dc214e4e84b57d9a",
        "domain": "numina",
        "question": "2. The sequence $\\left(a_{n}\\right)$ is defined by the following relations: $a_{1}=1, a_{2}=3, a_{n}=a_{n-1}-a_{n-2}+n$ (for $n \\geqslant 3$). Find $a_{1000}$.",
        "answer": "1002",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3983,
        "difficulty": null
      },
      {
        "id": "0e04b0bbda57aa318d7bf1fa603f6d8a5056787b3400773d1ffe72aaee875036",
        "domain": "numina",
        "question": "\\section*{Problem 14}\n\nA spy-plane circles point A at a distance \\(10 \\mathrm{~km}\\) with speed \\(1000 \\mathrm{~km} / \\mathrm{h}\\). A missile is fired towards the plane from A at the same speed and moves so that it is always on the line between A and the plane. How long does it take to hit?",
        "answer": "18\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 186935,
        "difficulty": null
      },
      {
        "id": "deb060995021daf6f6b61a529b6e2366fc06bf9cc1e1c8cf2ab17089f829d576",
        "domain": "numina",
        "question": "1. How many positive numbers are there among the 2014 terms of the sequence: $\\sin 1^{\\circ}, \\sin 10^{\\circ}$, $\\sin 100^{\\circ}, \\sin 1000^{\\circ}, \\ldots ?$",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19301,
        "difficulty": null
      },
      {
        "id": "0a333cf351efabcdcd1e020d90fdaf428f651cc03810eaa8940a69747b7aa21d",
        "domain": "numina",
        "question": "4. Determine the maximum value of the difference $x-y$ if\n\n$$\n2\\left(x^{2}+y^{2}\\right)=x+y\n$$",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 187592,
        "difficulty": null
      },
      {
        "id": "3eec67938c9c491f84e188393322c5f5e731482e3677cd7588149c681c6f11bd",
        "domain": "numina",
        "question": "13. Xiao Li and Xiao Zhang are running at a constant speed on a circular track. They start at the same time and place, with Xiao Li running clockwise and completing a lap every 72 seconds; Xiao Zhang running counterclockwise and completing a lap every 80 seconds. A quarter-circle arc interval is marked on the track, centered at the starting point. How many seconds do they spend in the marked interval simultaneously from the start until they both return to the starting point at the same time?",
        "answer": "46",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144079,
        "difficulty": null
      },
      {
        "id": "612412bbcf5779764b714dc6cf38a57828f2e26e8babcc67b6c92ae9a0061ae9",
        "domain": "numina",
        "question": "O esqueleto de um cubo $6 \\times 6 \\times 6$, formado por cubinhos $1 \\times 1 \\times 1$ é mostrado na figura.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_0308bd91a070e7a80796g-3.jpg?height=500&width=437&top_left_y=387&top_left_x=638)\n\nFigura 64.1\n\n(a) Quantos cubinhos formam este esqueleto?\n\n(b) É dado um cubo $7 \\times 7 \\times 7$ formado por cubinhos $1 \\times 1 \\times 1$. Quantos cubinhos devemos retirar para obter um esqueleto do cubo $7 \\times$ $7 \\times 7$.\n\n##",
        "answer": "56",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74269,
        "difficulty": null
      },
      {
        "id": "fd919d31c5cc11d91c3276836901b1de8073493344d9ec1a4068eca92c3b00f4",
        "domain": "numina",
        "question": "A train traveling between $A$ and $B$ arrives 20 minutes earlier in $B$ if its speed exceeds the scheduled speed by $5 \\mathrm{~km}$ per hour; however, it arrives 25 minutes late if its speed is $5 \\mathrm{~km}$ per hour less than the scheduled speed. What is the scheduled speed?",
        "answer": "45",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 172312,
        "difficulty": null
      },
      {
        "id": "af84493169a25d7813d10f66efdebc90d630eb8a5c4af4bd212ab969fea115ce",
        "domain": "numina",
        "question": "$12 \\cdot 60$ In a right-angled triangle with side lengths all being positive integers, one of the legs is 1575 less than the hypotenuse, and the other leg is less than 1991. Find the length of the hypotenuse of this right-angled triangle.\n(China Beijing High School Grade 1 Mathematics Competition, 1991)",
        "answer": "1799",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 95453,
        "difficulty": null
      },
      {
        "id": "6cfa62de8b851b892b738536c01a94f299e89f374f6e28dd2309493ed7db9ad7",
        "domain": "numina",
        "question": "Problem 9.3. Given a convex quadrilateral $ABCD$, $X$ is the midpoint of diagonal $AC$. It turns out that $CD \\parallel BX$. Find $AD$, if it is known that $BX=3, BC=7, CD=6$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_790dd471148872cd0846g-02.jpg?height=224&width=495&top_left_y=1341&top_left_x=479)",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 18648,
        "difficulty": null
      },
      {
        "id": "16b8742b0f1646cb34de0e0f80a9c4c978056676823cc660e215002e08c8e5c3",
        "domain": "numina",
        "question": "Task 2. Every student in the Netherlands receives a finite number of cards. Each card has a real number in the interval $[0,1]$ written on it. (The numbers on different cards do not have to be different.) Find the smallest real number $c>0$ for which the following holds, regardless of the numbers on the cards that everyone has received.\n\nEvery student whose sum of the numbers on the cards is at most 1000 can distribute the cards over 100 boxes so that the sum of the numbers in each box is at most $c$.",
        "answer": "11-\\frac{1}{91}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 85466,
        "difficulty": null
      },
      {
        "id": "d12944724e9158091e4bbe9a4bd65e8bf1ce733519fffa0cb5344a88f7582935",
        "domain": "numina",
        "question": "9. Given the sequence $\\left\\{a_{n}\\right\\}$ with the sum of the first $n$ terms being $S_{n}, a_{1}=\\frac{1}{5}$, and for any positive integers $m, n$, it holds that $a_{m+n}=a_{m} a_{n}$. If $S_{n}<a$ for any $n \\in \\mathbf{Z}_{+}$, then the minimum value of the real number $a$ is",
        "answer": "\\frac{1}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118726,
        "difficulty": null
      },
      {
        "id": "9f6f49b131a9aa56c6276fc86587b1cf0cf33e898e42a1d8241b72699a6e3fdc",
        "domain": "numina",
        "question": "50.What is the 1997th digit in the decimal expansion of the fraction $\\frac{1}{7}=0.142857 \\ldots ?$",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35808,
        "difficulty": null
      },
      {
        "id": "917ab841083514c7275fd67fdd3e33db3b5a832b4e2b239a002ce9d3c4b7a566",
        "domain": "numina",
        "question": "9. Given $f(x)=\\left\\{\\begin{array}{l}\\frac{3+x}{1+x^{2}}, 0 \\leq x \\leq 3, \\\\ \\frac{3}{5}, x>3 .\\end{array}\\right.$, the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $0<a_{n} \\leq 3, n \\in N^{*}$, and $\\sum_{i=1}^{2009} a_{i}=\\frac{2009}{3}$, if the inequality $\\sum_{i=1}^{2009} f\\left(a_{i}\\right) \\leq x-\\ln (x-p)$ holds for all $x \\in(p,+\\infty)$, find the minimum value of the real number $p$.",
        "answer": "6026",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 123331,
        "difficulty": null
      },
      {
        "id": "c97088780634f53efbda6e3237dab878251e3469921b5081a701e26ee30d79fd",
        "domain": "numina",
        "question": "3. Let the real numbers $x_{1}, x_{2}$, and $x_{3}$ be the roots of the equation $3 x^{3}-p x^{2}+28 x-p=0$, taking into account their possible multiplicities. Then $3 x^{3}-p x^{2}+28 x-p \\equiv 3\\left(x-x_{1}\\right)\\left(x-x_{2}\\right)\\left(x-x_{3}\\right)=$\n\n$$\n=3 x^{3}-3 x^{2}\\left(x_{1}+x_{2}+x_{3}\\right)+3 x\\left(x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}\\right)-3 x_{1} x_{2} x_{3},\n$$\n\nfrom which $x_{1}+x_{2}+x_{3}=p / 3, \\quad x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}=28 / 3$. Taking this into account, the target expression is transformed as follows:\n\n$$\n\\begin{aligned}\n&\\left(x_{1}-2\\right)^{2}+\\left(x_{2}-2\\right)^{2}+\\left(x_{3}-2\\right)^{2}=x_{1}^{2}+x_{2}^{2}+x_{3}^{2}-4\\left(x_{1}+x_{2}+x_{3}\\right)+12= \\\\\n&=\\left(x_{1}+x_{2}+x_{3}\\right)^{2}-2\\left(x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}\\right)-4\\left(x_{1}+x_{2}+x_{3}\\right)+12= \\\\\n&= \\frac{p^{2}}{9}-\\frac{56}{3}-\\frac{4 p}{3}+12=\\frac{p^{2}-12 p-60}{9} .\n\\end{aligned}\n$$\n\nNow let's plot the graph of the equation $3 x^{3}-p x^{2}+28 x-p=0$ on the plane $(x, p)$. Since the parameter enters this equation linearly, it is more convenient to solve it for $p$:\n\n$$\np\\left(x^{2}+1\\right)=3 x^{3}+28 x=3 x\\left(x^{2}+1\\right)+25 x \\Rightarrow p(x)=3 x+\\frac{25 x}{x^{2}+1}\n$$\n\nWe will find the set of values of the parameter $p$ for which the equation has three real roots. For this, we will investigate $p(x)$:\n\n$$\np^{\\prime}(x)=3+\\frac{25\\left(1-x^{2}\\right)}{\\left(x^{2}+1\\right)^{2}}=\\frac{3 x^{4}-19 x^{2}+28}{\\left(x^{2}+1\\right)^{2}}=\\frac{\\left(x^{2}-4\\right)\\left(3 x^{2}-7\\right)}{\\left(x^{2}+1\\right)^{2}}\n$$\n\nThe odd function $p(x)$ has minima at points $\\bar{x}_{2}=-\\frac{\\sqrt{21}}{3}, \\bar{x}_{4}=2$ and maxima at points $\\bar{x}_{1}=-2, \\bar{x}_{3}=\\frac{\\sqrt{21}}{3}$,\n\nwhere\n\n$$\np_{1}=p\\left(\\bar{x}_{1}\\right)=-16, p_{2}=p\\left(\\bar{x}_{2}\\right)=-\\frac{7 \\sqrt{21}}{2}\n$$\n\n$$\np_{3}=p\\left(\\bar{x}_{3}\\right)=\\frac{7 \\sqrt{21}}{2}, p_{4}=p\\left(\\bar{x}_{4}\\right)=16\n$$\n\nFor all $c \\in\\left(p_{2} ; p_{1}\\right) \\cup\\left(p_{4} ; p_{3}\\right)$, the line $p=c$ intersects the graph at three points, and the equation $3 x^{3}-p x^{2}+28 x-p=0$ has three distinct roots. For parameter values that coincide with one of $p_{1}, p_{2}, p_{3}$, or $p_{4}$ (on the boundaries of the intervals), there are two points of intersection, which corresponds to the case of multiple roots.\n\nThe quadratic trinomial $\\frac{p^{2}-12 p-60}{9}$, decreasing for $p6$, takes the smallest possible value on the set $\\left[p_{2} ; p_{1}\\right] \\cup\\left[p_{4} ; p_{3}\\right]$ when $p=p_{4}=16$.\n\nThus, under the conditions of the problem,\n\n$$\n\\min \\left(\\left(x_{1}-2\\right)^{2}+\\left(x_{2}-2\\right)^{2}+\\left(x_{3}-2\\right)^{2}\\right)=\\frac{16^{2}-12 \\cdot 16-60}{9}=\\frac{4}{9}\n$$",
        "answer": "\\frac{4}{9}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 2339,
        "difficulty": null
      },
      {
        "id": "243ea3ca4ff9d04b2c9e7480c6610d2f19fe5d2cf0949efd88f77315c78f03eb",
        "domain": "numina",
        "question": "A game is composed of the following rules:\n\ni) In each round, a common unbiased die is rolled.\n\nii) If the number 3 comes up, then player $A$ wins.\n\niii) If one of the numbers in the set $\\{4,5,6\\}$ comes up, then player $B$ wins.\n\niv) If one of the numbers in the set $\\{1,2\\}$ comes up, then the die is rolled again until it results in 3 or 4 or 5 or 6.\n\nWhat is the probability that player $B$ wins?",
        "answer": "\\frac{3}{4}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73706,
        "difficulty": null
      },
      {
        "id": "3783b7827ac83b39c2e24ab146c88bffac263893d86b39e383150f8733625eaf",
        "domain": "numina",
        "question": "1.5 Remove the multiples of 3 and 4 from the sequence of natural numbers $1,2,3,4, \\cdots$, but retain all multiples of 5 (for example, 15 and 20 are not removed). After removing, write the remaining numbers in a sequence: $A_{1}=1, A_{2}=2, A_{3}=5, A_{4}=7, \\cdots$ Find the value of $A_{1988}$.\n(Advanced Math Test for Science Experimental Class Admission, 1988)",
        "answer": "3314",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 135214,
        "difficulty": null
      },
      {
        "id": "06800e1b06b4258b5ad76a71a37241545896649c59ecb2ddebba176ea23363d8",
        "domain": "numina",
        "question": "16. If an internal point of some $n$-sided prism is connected to all its vertices, then we obtain $n$ quadrilateral pyramids with a common vertex at this point, the bases of which are the lateral faces of the prism. Find the ratio of the sum of the volumes of these pyramids to the volume of the given prism.",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44465,
        "difficulty": null
      },
      {
        "id": "b31821b55ecacba7a11515cf095057f5398a5cd7e418236716c56d8edf1577ea",
        "domain": "numina",
        "question": "13. If the six-digit number $a 2016 b$ is divisible by 12, then the number of such six-digit numbers is $\\qquad$.",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136894,
        "difficulty": null
      },
      {
        "id": "accfac8e0e2de02ed46520b255e309c447be25fedba9e5c653829bb88842fd80",
        "domain": "numina",
        "question": "Problem 10.5. A circle $\\omega$ is inscribed in trapezoid $A B C D$, and $L$ is the point of tangency of $\\omega$ and side $C D$. It is known that $C L: L D=1: 4$. Find the area of trapezoid $A B C D$, if $B C=9$, $C D=30$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c47c8a04f76ef67ab4e5g-35.jpg?height=305&width=407&top_left_y=231&top_left_x=517)",
        "answer": "972",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17894,
        "difficulty": null
      },
      {
        "id": "1b15f140f1c6fad5bf093e6c75d0fdec357819d720f3fcaecf747dc8461c1633",
        "domain": "numina",
        "question": "Given a rectangle inscribed in a circle, we extend it to an octagon, four of whose vertices coincide with the vertices of the rectangle, and the other four are the intersection points of the perpendicular bisectors of the rectangle's sides and the circle. How should we choose the rectangle so that the ratio of the area of the octagon to the area of the rectangle is minimized?",
        "answer": "\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166667,
        "difficulty": null
      },
      {
        "id": "ad5d96f3b94d3705737a346157eb67f17cf39f8dee33a3759b4882e74018e477",
        "domain": "numina",
        "question": "5. Ari is watching several frogs jump straight towards him from the same place. Some frogs are larger, and some are smaller. When the larger frog makes 7 equal jumps, it is still $3 \\mathrm{~cm}$ away from Ari. When the smaller frog makes 10 equal jumps, it is still $1 \\mathrm{~cm}$ away from Ari. If the frogs are initially more than $2 \\mathrm{~m}$ away from Ari, what is the smallest possible length of the smaller frog's jump? The length of each jump in centimeters is a natural number.\n\nNo pocket calculators or any reference materials are allowed.\n\n## Ministry of Science and Education of the Republic of Croatia Agency for Education and Education Croatian Mathematical Society\n\n## County Competition in Mathematics February 26, 2024.  5th grade - elementary school\n\nEach task is worth 10 points. In addition to the final result, the procedure is also scored. To earn all points, it is necessary to find all solutions and confirm that there are no others, write down the procedure, and justify your conclusions.",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86488,
        "difficulty": null
      },
      {
        "id": "4626f698f6ec2e3131ea237fb53911bdfaab227738048b9dafd097783201418f",
        "domain": "numina",
        "question": "Connect the midpoints of the sides of a rectangle with the opposite vertices. What fraction of the area of the original rectangle is the area of the octagon determined by these eight lines?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{1}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171562,
        "difficulty": null
      },
      {
        "id": "ba89f8eee4685623cef8bc5b47fdff271fee55fdce73e5d9f4aaf6d86ddce71f",
        "domain": "numina",
        "question": "(1) If $4^{a}-3 a^{b}=16, \\log _{2} a=\\frac{a+1}{b}$, then $a^{b}=$",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136892,
        "difficulty": null
      },
      {
        "id": "f09461ef5d45a54e2182015dd8b60131655bb12c8ebf781d5f1eaf2e1784664c",
        "domain": "numina",
        "question": "24. Let $a$ and $b$ be two real numbers. If the equation $a x+(b-3)=(5 a-1) x+3 b$ has more than one solution, what is the value of $100 a+4 b$ ?",
        "answer": "19",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160892,
        "difficulty": null
      },
      {
        "id": "f71135cd5bf7c4acf6257ce5b6649e3ba870cd5f5c8e344e378faee52eeb54b1",
        "domain": "numina",
        "question": "13.333. A ball falls from a height of 2 m 43 cm and, upon hitting the ground, bounces back up, each time reaching $2 / 3$ of the height from which it falls again. After how many bounces will the ball rise to a height of 32 cm?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49419,
        "difficulty": null
      },
      {
        "id": "db069b7c266a12bfb2a18d46b8f8de9eecad468e36e392833378b5aefb0fcd8e",
        "domain": "numina",
        "question": "90. Use the digits 1 to 9 to form a two-digit number, a three-digit number, and a four-digit number, with each digit used only once. The maximum sum of these three numbers is $\\qquad$ .",
        "answer": "10656",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 103569,
        "difficulty": null
      },
      {
        "id": "00475475382d123e54cc9c757a992c8f716228ef843263359422207433a143d0",
        "domain": "numina",
        "question": "53. A snail is crawling on a plane, turning $90^{\\circ}$ after each meter of its path. What is the maximum distance from the starting point it could be after crawling 300 m, having made a total of 99 left and 200 right turns?",
        "answer": "100\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 37285,
        "difficulty": null
      },
      {
        "id": "b09b18e9d7911404dcd48cf199bf4a4eb48f2330e5988a014bcbd78df69e8186",
        "domain": "numina",
        "question": "1. In the diagram, $\\triangle A B C$ has angles that measure $60^{\\circ}$, $(5 x)^{\\circ}$ and $(3 x)^{\\circ}$. What is the value of $x$ ?",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144666,
        "difficulty": null
      },
      {
        "id": "2cd575d4a6a691f718b263dd0c73ca3cfb9571f31a0698b6f7dd4841f16aa5e5",
        "domain": "numina",
        "question": "7. For an arithmetic sequence with real number terms, the common difference is 4, and the square of the first term plus the sum of the remaining terms does not exceed 100. The maximum number of terms in such a sequence is $\\qquad$.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142450,
        "difficulty": null
      },
      {
        "id": "7946675062cea63bc0920d4ef000ec086be9545841f60ebc6332d3bb3fba320d",
        "domain": "numina",
        "question": "NT 3. Find the largest integer $k(k \\geq 2)$, for which there exists an integer $n(n \\geq k)$ such that from any collection of $n$ consecutive positive integers one can always choose $k$ numbers, which verify the following conditions:\n\n1. each chosen number is not divisible by 6 , by 7 and by 8 ;\n2. the positive difference of any two different chosen numbers is not divisible by at least one of the numbers 6,7 or 8 .",
        "answer": "108",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 228,
        "difficulty": null
      },
      {
        "id": "03c81fb1eb2c56c4a38b8a5a1114e2f5ad8f957be0872ad9e38c8e1d237fbdb6",
        "domain": "numina",
        "question": "32. (USA 3) The vertex $A$ of the acute triangle $A B C$ is equidistant from the circumcenter $O$ and the orthocenter $H$. Determine all possible values for the measure of angle $A$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 158267,
        "difficulty": null
      },
      {
        "id": "6ddff69a72a4c8c94e7b3e18daada4e853ec657e055b7ddede55ec5aee2a6ef0",
        "domain": "numina",
        "question": "4. The real numbers $a, b, c, d$ are positive and satisfy $(a+c)(b+d)=a c+b d$. Find the minimum of\n\n$$\n\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}\n$$\n\nAnswer: 8.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91175,
        "difficulty": null
      },
      {
        "id": "8d85b5ca41af76f7ec0884dc6bd2cb4865628e365cb04611e9178c539563d607",
        "domain": "numina",
        "question": "Example 13 Let $a, b$ be non-zero complex numbers, and $\\frac{a}{b}$ is not a real number. Define:\n$$\n\\begin{array}{l}\nL_{a, b}=\\{r a+s b \\mid r, s \\in \\mathbf{Z}\\}, \\\\\nR_{a, b}=\\left\\{z \\mid z \\text { is a non-zero complex number, and } L_{a, b}=L_{z a, z}\\right\\} .\n\\end{array}\n$$\n\nTry to find the maximum number of elements in the set $R_{a, b}$ as $a, b$ vary.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 132141,
        "difficulty": null
      },
      {
        "id": "a304681d898920fde9fd5dcc748d73779794314fbb6a09166d9349d4ed474dcb",
        "domain": "numina",
        "question": "Problem 5. A square with an integer side length was cut into 2020 squares. It is known that the areas of 2019 squares are 1, and the area of the 2020th square is not equal to 1. Find all possible values that the area of the 2020th square can take. In your answer, provide the smallest of the obtained area values.",
        "answer": "112225",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 4978,
        "difficulty": null
      },
      {
        "id": "6e1ed39b0b8c42082b654497e2259a2733a9d636f53b100cf22104accf2e94e4",
        "domain": "numina",
        "question": "10-3. Point $O$ is the center of the circle. What is the value of angle $x$ in degrees?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d9f67fd4a24dc8d44db8g-25.jpg?height=488&width=870&top_left_y=2269&top_left_x=593)",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 18700,
        "difficulty": null
      },
      {
        "id": "a31af3fb9ba34067eabb8a361320d97c58181d9a13be26aed762a2b6a8e319a7",
        "domain": "numina",
        "question": "1. Let positive integers $a_{1}, a_{2}, \\cdots, a_{31}, b_{1}, b_{2}, \\cdots, b_{31}$ satisfy\n$$\n\\begin{array}{l}\n\\text { (1) } a_{1}<a_{2}<\\cdots<a_{31} \\leqslant 2015, \\\\\nb_{1}<b_{2}<\\cdots<b_{31} \\leqslant 2015 ; \\\\\n\\text { (2) } a_{1}+a_{2}+\\cdots+a_{31}=b_{1}+b_{2}+\\cdots+b_{31} \\text {. } \\\\\n\\text { Find } S=\\left|a_{1}-b_{1}\\right|+\\left|a_{2}-b_{2}\\right|+\\cdots+\\left|a_{31}-b_{31}\\right|\n\\end{array}\n$$\n\nthe maximum value.\n(Supplied by He Yijie)",
        "answer": "30720",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100871,
        "difficulty": null
      },
      {
        "id": "7fe8e35d277fd58c9bc90227a2ff05e3e0c3c565308450b80be8760a76f27eee",
        "domain": "numina",
        "question": "## Task A-1.2.\n\nAll natural numbers from 1 to some number were written on a board. After one of the numbers was erased, the arithmetic mean of the remaining numbers on the board is $\\frac{673}{18}$. Which number was erased?",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87771,
        "difficulty": null
      },
      {
        "id": "58429753b04f70e64680f2fe29bd6e713ee5cc2858c39bd2dfed40629907b69f",
        "domain": "numina",
        "question": "[Pythagorean Theorem (direct and inverse).] [ Theorem on the sum of the squares of the diagonals ]\n\nThe height of a rhombus, drawn from the vertex of the obtuse angle, divides its side into segments of length $a$ and $b$. Find the diagonals of the rhombus.",
        "answer": "65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65303,
        "difficulty": null
      },
      {
        "id": "8c9209a32650936e39c160e02c3f767604b6571a47c35f226b118e2c05dc67bf",
        "domain": "numina",
        "question": "1. The numerical sequence $\\left\\{a_{n}\\right\\}_{n=1}^{\\infty}$ is defined such that $a_{1}=\\log _{2}\\left(\\log _{2} f(2)\\right), \\quad a_{2}=$ $\\log _{2}\\left(\\log _{2} f(f(2))\\right), \\ldots, a_{n}=\\log _{2}(\\log _{2} \\underbrace{f(f(\\ldots f}_{n}(2)))), \\ldots$, where $f(x)=x^{x}$. Determine the index $n$ for which $a_{n}=2059+2^{2059}$.\n\n(12 points)",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10606,
        "difficulty": null
      },
      {
        "id": "321d5f6d6fc81dfa8515df45c5bda833b9c3fa6c869172c9e94c0324329f9a28",
        "domain": "numina",
        "question": "3 [ Perpendicularity of a line and a plane (other).]\n\nThe base of the pyramid is an equilateral triangle with a side length of 6. One of the lateral edges is perpendicular to the base plane and equals 4. Find the radius of the sphere circumscribed around the pyramid.\n\n#",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66179,
        "difficulty": null
      },
      {
        "id": "f65354789a815542a4abf4aef0307902390fb352db0e4a64403124d9d6617848",
        "domain": "numina",
        "question": "3. [5] Suppose\n$$\nh \\cdot a \\cdot r \\cdot v \\cdot a \\cdot r \\cdot d=m \\cdot i \\cdot t=h \\cdot m \\cdot m \\cdot t=100 \\text {. }\n$$\n\nFind $(r \\cdot a \\cdot d) \\cdot(t \\cdot r \\cdot i \\cdot v \\cdot i \\cdot a)$.",
        "answer": "10000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151908,
        "difficulty": null
      },
      {
        "id": "51c1a50bbfbe964405f530030f68ed2c69792c9472a6daeec47079b0775bf5a2",
        "domain": "numina",
        "question": "Problem 11.8. In a $28 \\times 35$ table, some $k$ cells are painted red, another $r-$ in pink, and the remaining $s-$ in blue. It is known that\n\n- $k \\geqslant r \\geqslant s$\n- each boundary cell has at least 2 neighbors of the same color;\n- each non-boundary cell has at least 3 neighbors of the same color.\n\nWhat is the smallest value that the quantity $k-s$ can take?\n\n(A cell is called boundary if it is adjacent to the edge of the table. Neighbors are cells that share a common side.)",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13695,
        "difficulty": null
      },
      {
        "id": "aa5ea893fc7da1fc237d490d7bd2dc12d3607367742250fdb9f7b6b9512636b2",
        "domain": "numina",
        "question": "9.6. Petya and Vasya came up with ten quadratic trinomials. Then Vasya sequentially named consecutive natural numbers (starting from some number), and Petya substituted each named number into one of the trinomials of his choice and wrote down the obtained values on the board from left to right. It turned out that the numbers written on the board form an arithmetic progression (in this exact order). What is the maximum number of numbers Vasya could have named?\n\n(A. Golev)",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14830,
        "difficulty": null
      },
      {
        "id": "a206da9d54812bff71ec029ea324f8670fd26acca69e50e1e22310e51c96b97d",
        "domain": "numina",
        "question": "Example 6 Let $f(x)$ be an even function defined on $\\mathbf{R}$, whose graph is symmetric about the line $x=1$. For any $x_{1}, x_{2} \\in \\left[0, \\frac{1}{2}\\right]$, we have $f\\left(x_{1}+x_{2}\\right)=f\\left(x_{1}\\right) \\cdot f\\left(x_{2}\\right)$. Also, $f(1)=a>0$.\n(1) Find $f\\left(\\frac{1}{2}\\right)$ and $f\\left(\\frac{1}{4}\\right)$;\n(2) Prove that $f(x)$ is a periodic function;\n(3) Let $a_{n}=f\\left(2 n+\\frac{1}{2 n}\\right)$, find $\\lim _{n \\rightarrow \\infty}\\left(\\ln a_{n}\\right)$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131425,
        "difficulty": null
      },
      {
        "id": "b18c25a44e47483c034d36448ee307ad7989fd6f438e51b3af42899900d8dc6e",
        "domain": "numina",
        "question": "8.2. Little kids were eating candies. Each one ate 7 candies less than all the others together, but still more than one candy. How many candies were eaten in total?",
        "answer": "21",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12321,
        "difficulty": null
      },
      {
        "id": "fab4d0c159cbd0b9fbdfaa9538641cdc48050b0db4a436456c647238d679cf97",
        "domain": "numina",
        "question": "4. As shown in Figure 9, the side length of square $A B C D$ is 10, and points $P$ and $Q$ are on the line segment connecting the midpoints of $A D$ and $B C$. Connecting $P A$, $P C$, $Q A$, and $Q C$, the square is divided into three parts of equal area. Find the length of line segment $P Q$.",
        "answer": "\\frac{20}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143015,
        "difficulty": null
      },
      {
        "id": "437f16f4354295f35ac0152acae3194fe7cb0cf3c38e7edc4613f1a4264e9222",
        "domain": "numina",
        "question": "5. The number 2015 was divided into 12 addends, after which all numbers that can be obtained by adding some of these addends (from one to nine) were written down. What is the minimum number of numbers that could have been written down?",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 21365,
        "difficulty": null
      },
      {
        "id": "1974dfa34d2f7ddabf5ba8cc03927ddbe31f0e97e253654f282cd9cf99ebe52b",
        "domain": "numina",
        "question": "2. (15 points) Two medians of a triangle, equal to 18 and 24, are perpendicular to each other. Find the length of the third median of this triangle.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21900,
        "difficulty": null
      },
      {
        "id": "982086c175284784da692d1ce8346c17903c4a9a0621f0acbaddcd58cf9437b4",
        "domain": "numina",
        "question": "13. Lucio buys a table in the shape of a regular hexagon and a rectangular tablecloth with an area of $1024 \\mathrm{~cm}^{2}$ that has the shorter side equal to the side of the hexagon and covers exactly the portion of the table between two opposite sides. What is the area of the table, expressed in $\\mathrm{cm}^{2}$?",
        "answer": "1536",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 181309,
        "difficulty": null
      },
      {
        "id": "37484b248c3984865ef3873e7939a8298c57eb59949ada0b3b1e0fdfee307be5",
        "domain": "numina",
        "question": "(11 Let $\\sec x+\\tan x=\\frac{22}{7}, \\csc x+\\cot x=\\frac{m}{n}(m, n$ have no common divisor greater than 1), then $m+n=$ $\\qquad$",
        "answer": "44",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128779,
        "difficulty": null
      },
      {
        "id": "e36f15d4beb543fb64cf63e3bf53f2006c3cd4946ef175e9911554200938177b",
        "domain": "numina",
        "question": "40. Two different natural numbers $a, b$, their least common multiple is 64, then their greatest common divisor has\n$\\qquad$ possible situations.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147092,
        "difficulty": null
      },
      {
        "id": "2e5e2cf1d9a986e8beba6fa124d8dfb03811feac6099731394852f280ac5d955",
        "domain": "numina",
        "question": "54. The task is very difficult-\n\nNot everyone can find:\n\nWhat does the star,\n\nBicycle and hedgehog equal?\n\n| BICYCLE | HEDGEHOG | 7 |\n| :---: | :---: | :---: |\n| + STAR | HEDGEHOG | 4 |\n| 6 | BICYCLE HEDGEHOG |  |\n| 1 BICYCLE | 0 | STAR |\n\nDecode the rebuses:",
        "answer": "1503",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 39363,
        "difficulty": null
      },
      {
        "id": "7917454d8a71b8f8ca7525ef50020734a69d9ed2fe0c9c23f74c7f6b084e4944",
        "domain": "numina",
        "question": "In the figure below, $A B C D$ and $E F G H$ are rectangles with parallel sides. Given that $A E=10$, $B F=20$ and $D H=30$, determine the length of the segment $C G$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-29.jpg?height=471&width=751&top_left_y=581&top_left_x=755)",
        "answer": "20\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73961,
        "difficulty": null
      },
      {
        "id": "2d1236e7eebc6cb908a1e46ad97652601c83f72cabd74cabaf8a004d8fc20d4e",
        "domain": "numina",
        "question": "91. A pasture is full of grass, which grows at a uniform rate every day. 17 cows can finish eating the grass in 30 days, 19 cows can finish in 20 days. Now, a certain number of cows eat for 5 days, then 3 cows are sold, and the remaining cows finish eating the grass in 2 more days. How many cows were there initially eating the grass (grass grows uniformly)?\n\n原有 $\\qquad$头牛吃草\n\nInitially, there were $\\qquad$ cows eating the grass.",
        "answer": "31",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141832,
        "difficulty": null
      },
      {
        "id": "2bafb334e0f1d1f104bc553fb7957fbe2117dbbcb7c73d8eba1a6d7bd583d651",
        "domain": "numina",
        "question": "14. Let $a, b, c$ be positive real numbers such that\n$$\n\\left\\{\\begin{array}{l}\na^{2}+a b+b^{2}=25, \\\\\nb^{2}+b c+c^{2}=49, \\\\\nc^{2}+a+a^{2}=64 .\n\\end{array}\\right.\n$$\n\nFind $(a+b+c)^{2}$.",
        "answer": "129",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160661,
        "difficulty": null
      },
      {
        "id": "dc86b1a28060c46a89adc05695fb3930fa5c0bf84541a8c4ba4542a3a09ba126",
        "domain": "numina",
        "question": "96. A train takes 26 seconds to fully pass a bridge that is 335 meters long, and it takes 63 seconds to pass a bridge that is 1075 meters long at the same speed. The length of this train is _ meters.",
        "answer": "185",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115626,
        "difficulty": null
      },
      {
        "id": "157cef4367496ea5e11e8368bf7257096bfde4bd8cc41db9acf677bcb9feba00",
        "domain": "numina",
        "question": "10,11 |\n\nLet $x, y, z$ be positive numbers and $x y z(x+y+z)=1$. Find the minimum value of the expression $(x+y)(x+z)$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 52738,
        "difficulty": null
      },
      {
        "id": "b105e5af771fec324ce1a58b662fbc7c26c5494d355a2d3c769505c854eaed06",
        "domain": "numina",
        "question": "3. Equilateral $\\triangle A B C$ and square $A B D E$ share a common side $A B$, the cosine of the dihedral angle $C-A B-D$ is $\\frac{\\sqrt{3}}{3}, M, N$ are the midpoints of $A C, B C$ respectively, then the cosine of the angle formed by $E M, A N$ is $\\qquad$ .",
        "answer": "\\frac{1}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144656,
        "difficulty": null
      },
      {
        "id": "6eee5a9ad7c1f1c5797012a2d616cecb46b0807ea83a985c0e8ea6f699bc57c4",
        "domain": "numina",
        "question": "2. Point $B$ is internal to segment $\\overline{A C}$. In the same half-plane relative to line $A C$, equilateral triangles $\\triangle A B M$ and $\\triangle B C N$ are constructed. Lines $A N$ and $C M$ intersect at point $L$. Determine the angle $\\measuredangle C L N$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189345,
        "difficulty": null
      },
      {
        "id": "14ac1861ee8bf0a79deb2833eaa25628ac94bd5895048075a052a5ccdb3ca0db",
        "domain": "numina",
        "question": "## 10.5 Find:\n\na) the real part of the complex number $z=\\frac{(2-i)^{3}}{3+4 i}$;\n\nb) the imaginary part of the complex number $z=\\left(\\frac{1+i}{1-i}\\right)^{11}$.\n\nPerform the operations (10.6-10.8):",
        "answer": "-1.52",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49892,
        "difficulty": null
      },
      {
        "id": "3b26c57fb4fb2650871bb645ac3e8f29b3f453928325ad349532b8c86f561ba1",
        "domain": "numina",
        "question": "## Task B-1.2.\n\nWith which digit does the number $2^{2022}+3^{2022}+7^{2022}$ end?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88452,
        "difficulty": null
      },
      {
        "id": "d67b719b2428a46012411c9ad098b345ae1abce69450981d182194949c4aca2f",
        "domain": "numina",
        "question": "Example 8 Determine the maximum value of $m^{2}+n^{2}$, where $m, n \\in\\{1,2, \\cdots, 2005\\}$ satisfy $\\left(n^{2}+2 m n-2 m^{2}\\right)^{2}=1$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "702036",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109697,
        "difficulty": null
      },
      {
        "id": "f16a1fdcd95737c4e9aa8122b8762e5f6e4dbe935a3be3b9f52bd57af903ed2b",
        "domain": "numina",
        "question": "## Task 1 - 290811\n\nOn a bottle of standard 40 percent acetic acid, the following instructions are given: \"The contents of this bottle ( $200 \\mathrm{ml}$ ), mixed with $800 \\mathrm{ml}$ of water, yield a ten percent vinegar.\"\n\nIs this correct?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184257,
        "difficulty": null
      },
      {
        "id": "e44e1831a7bd462da9c12378065450717117174d596e953934416e032b79ff01",
        "domain": "numina",
        "question": "9. Given the function $f(x)=a x^{3}+b x^{2}+c x+d(a \\neq 0)$, when $0 \\leqslant x \\leqslant 1$, $\\left|f^{\\prime}(x)\\right| \\leqslant 1$, try to find the maximum value of $a$.",
        "answer": "\\frac{8}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 103462,
        "difficulty": null
      },
      {
        "id": "fac1f819ff16a345734f2ec4e77533f3b2751377932f79ea6984c5ba446ccbbb",
        "domain": "numina",
        "question": "4. Find all positive real numbers $a$ such that there exists a positive integer $n$ and $n$ pairwise disjoint infinite sets $A_{1}, A_{2}, \\cdots, A_{n}$ satisfying $A_{1} \\cup A_{2} \\cup \\cdots \\cup A_{n}=\\mathbf{N}^{*}$, and for any two numbers $b>c$ in each $A_{i}$, we have $b-c \\geqslant a^{i}$.",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119572,
        "difficulty": null
      },
      {
        "id": "b6835a0753365a52aade93efaa84705b5065f573e310de59581c305bb5bcc423",
        "domain": "numina",
        "question": "23. Container A contains 2 liters of sulfuric acid solution with a concentration of $30 \\%$, and container B contains 3 liters of sulfuric acid solution with a concentration of $10 \\%$. Now, sulfuric acid solutions with concentrations of $6 \\%$ and $22 \\%$ are being added to containers A and B, respectively, at rates of 0.1 liters per minute and 0.3 liters per minute. After $\\qquad$ minutes, the concentrations of the sulfuric acid solutions in containers A and B will be equal.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139277,
        "difficulty": null
      },
      {
        "id": "70c49b80af55e8717e3f80453a1e0c7fa521dd3e43f341c8fe724f4149d76a17",
        "domain": "numina",
        "question": "$6 \\cdot 18$ Let $N$ be the set of positive integers, and define the function $f$ on $N$ as follows:\n(i) $f(1)=1, f(3)=3$;\n(ii) For $n \\in N$, we have\n$$\n\\begin{array}{l}\nf(2 n)=f(n), \\\\\nf(4 n+1)=2 f(2 n+1)-f(n), \\\\\nf(4 n+3)=3 f(2 n+1)-2 f(n) .\n\\end{array}\n$$\n\nFind all $n$ such that $n \\leqslant 1988$ and $f(n)=n$.",
        "answer": "92",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122781,
        "difficulty": null
      },
      {
        "id": "d8002bc2bea4882524c47daccdb5e22dcebc0ad6f2dea81faca55a1136512d97",
        "domain": "numina",
        "question": "3. Six children are sledding in a train formation down a hill. In how many different ways can they slide down if one of the children, who has peculiarities, believes that it is contraindicated for him to sit in even-numbered positions.",
        "answer": "360",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20362,
        "difficulty": null
      },
      {
        "id": "db3ca624f4bbe147833e7997a700f6a302acf83756f21387d53abf1e42b8ac6b",
        "domain": "numina",
        "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhat is greater: $1234567 \\cdot 1234569$ or $1234568^{2}$ ?",
        "answer": "1234568^{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 69030,
        "difficulty": null
      },
      {
        "id": "53201c2f7c202d764795f4ffca94e70f2b1bc1cf2f42c96f355935dd39cffa6b",
        "domain": "numina",
        "question": "8,9 trapezoids are constructed with circles on their diameters, intersecting at points $M$ and $N$. Chord $M N$ intersects the base $A D$ at point $E$. Find the ratio $A E: E D$.\n#",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57850,
        "difficulty": null
      },
      {
        "id": "a5f00ec2a145cd60fe338ae6245ae3f993aff01aa7193216fb483eab132392a2",
        "domain": "numina",
        "question": "Problem 10. (4 points)\n\nTo buy new headphones costing 275 rubles, Katya decided to save money on sports activities. Until now, she has been buying a single-visit ticket to the swimming pool, including a visit to the sauna for 250 rubles, to warm up. However, summer has arrived, and the need to visit the sauna has disappeared. Visiting only the swimming pool costs 200 rubles more than visiting the sauna. How many times does Katya need to visit the swimming pool without the sauna to save enough to buy the headphones? (Provide the answer as a whole number, without spaces or units of measurement.)\n\nAnswer: 11.\n\n## Comment:",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11524,
        "difficulty": null
      },
      {
        "id": "fff2c865e17858cda80616aba329ec7291e58d639618aa0efe065192454c3d00",
        "domain": "numina",
        "question": "12. Let the set $M=\\{1,2,3,4,5,6,7,8,9,10\\}, A=\\left\\{(x, y, z) \\mid x, y, z \\in M\\right.$, and $\\left.9 \\mid\\left(x^{3}+y^{3}+z^{3}\\right)\\right\\}$, then the number of elements in set $A$ is $\\qquad$.",
        "answer": "243",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94975,
        "difficulty": null
      },
      {
        "id": "1b7edeadb1e79df07540db5280b4430d4943912899036e24ec1aece12191ced6",
        "domain": "numina",
        "question": "9. The decagon $A B C D P Q R S T U V W$ has two parallel bases: square $A B C D$ and octagon $P Q R S T U V W$, all angles of which are equal, and eight lateral faces: triangles $A P Q, B R S, C T U, D V W$ and rectangles $D A P W, A B R Q, B C T S$ and $C D V U$. It is known that the area of the section of the decagon by a plane passing through points $A, S$ and $U$ is $\\frac{1372}{585},|A B|=1 / 3,|P Q|=2 \\sqrt{2} / 3$. Find the distance between the bases of the decagon. Write the found value in the answer, rounding it if necessary.",
        "answer": "2.65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5220,
        "difficulty": null
      },
      {
        "id": "4ef6607330bd051334dec2ef07e36ccde712e356eead3c9032ff148b1d7241b1",
        "domain": "numina",
        "question": "C4. Players $A$ and $B$ play a game with $N \\geq 2012$ coins and 2012 boxes arranged around a circle. Initially $A$ distributes the coins among the boxes so that there is at least 1 coin in each box. Then the two of them make moves in the order $B, A, B, A, \\ldots$ by the following rules:\n- On every move of his $B$ passes 1 coin from every box to an adjacent box.\n- On every move of hers $A$ chooses several coins that were not involved in $B$ 's previous move and are in different boxes. She passes every chosen coin to an adjacent box.\n\nPlayer A's goal is to ensure at least 1 coin in each box after every move of hers, regardless of how $B$ plays and how many moves are made. Find the least $N$ that enables her to succeed.",
        "answer": "4022",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153816,
        "difficulty": null
      },
      {
        "id": "f59627191c5c99ddff036e2372ff4b0919d8cb8248a7a873fa97cb099b1e9624",
        "domain": "numina",
        "question": "45th Putnam 1984 Problem A5 V is the pyramidal region x, y, z >= 0, x + y + z <= 1. Evaluate ∫ V x y 9 z 8 (1 - x - y - z) 4 dx dy dz. Solution",
        "answer": "\\frac{9!8!4!}{25!}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 149066,
        "difficulty": null
      },
      {
        "id": "833c3f993d5c6b5f4d6b86d5b9cf085426f8ce1ab2f431eeef8c1582e6d21744",
        "domain": "numina",
        "question": "Moskvitin N.A.\n\nIn triangle $A B C$, $A B = B C$. A perpendicular $E D$ is dropped from point $E$ on side $A B$ to $B C$. It turns out that $A E = E D$. Find the angle $D A C$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60283,
        "difficulty": null
      },
      {
        "id": "11bce79a01fe8144b2482e949aa1339be9159ed9a616f468f42303673088f2f3",
        "domain": "numina",
        "question": "6. How many $3 \\times 5$ rectangular pieces of cardboard can be cut from a $17 \\times 22$ rectangular piece of cardboard, when the amount of waste is minimised?",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153902,
        "difficulty": null
      },
      {
        "id": "678689f092e646efc47210764bfdea13286f96a24e6e98fb81157e7fa33b6908",
        "domain": "numina",
        "question": "Old cobbler Karl made boots and sent his son Hans to the market to sell them for 25 talers. At the market, two disabled men (one without a left leg, the other without a right leg) approached the boy and asked him to sell them each a boot. Hans agreed and sold each boot for 12.5 talers.\n\nWhen the boy came home and told his father everything, Karl decided that the disabled men should have been sold the boots cheaper - each for 10 talers. He gave Hans 5 talers and told him to return 2.5 talers to each disabled man.\n\nWhile the boy was looking for the disabled men at the market, he saw that sweets were being sold, couldn't resist, and spent 3 talers on candies. After that, he found the disabled men and gave them the remaining money - one taler each. Returning home, Hans realized how badly he had acted. He told everything to his father and asked for forgiveness. The cobbler got very angry and punished his son by putting him in a dark storeroom.\n\nSitting in the storeroom, Hans thought about it. It turned out that since he had returned one taler to each, the disabled men had paid 11.5 talers for each boot:\n\n$12.5-1=11.5$. So, the boots cost 23 talers: $2 \\cdot 11.5=23$. And Hans had spent 3 talers on candies, so in total it comes to 26 talers:\n\n$23+3=26$. But there were only 25 talers! Where did the extra taler come from?",
        "answer": "20",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56295,
        "difficulty": null
      },
      {
        "id": "3b9b2a18418d18ac451521677bb5d55e37831cd5650206a8b6c7c5239503d7a8",
        "domain": "numina",
        "question": "3. Find the number of natural numbers $k$, not exceeding 454500, such that $k^{2}-k$ is divisible by 505.",
        "answer": "3600",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 3197,
        "difficulty": null
      },
      {
        "id": "9078b04fd0fa43ef9566ce6ba3611c8bfe923ff2c99a00a7514978f1d6fda012",
        "domain": "numina",
        "question": "6. (7 points) Several sages lined up in a column. All of them wore either black or white caps. It turned out that among any 10 consecutive sages, there were an equal number of sages with white and black caps, while among any 12 consecutive sages - not an equal number. What is the maximum number of sages that could be\n\nAnswer: 15 sages.",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12551,
        "difficulty": null
      },
      {
        "id": "642a4e26596b8446c73d3354546d36149a187a39169c5b177856b146fa04fc50",
        "domain": "numina",
        "question": "Let's share a thought: divide every natural number by the sum of the squares of its digits (for single-digit numbers, divide by the square of the number).\n\nIs there a smallest quotient among the quotients obtained this way, and if so, which one is it?",
        "answer": "\\frac{1}{9}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169048,
        "difficulty": null
      },
      {
        "id": "039156dadd54db9f3ecb3f06e84466e86477e68c7a107aacc208d1c730a63f83",
        "domain": "numina",
        "question": "7. Anchurian Checkers ${ }^{1}$ (from 7th grade, 1 point). A tournament of Anchurian Checkers is held in several rounds. If an even number of players participate in a round, they are divided into random pairs. If the number of players is odd, random pairs are selected by lot, and one player remains without a pair and does not participate in the round. A draw cannot occur; the loser in each pair is eliminated from the tournament, while the winners and the player without a pair, if there is one, advance to the next round, which is conducted according to the same rules. This continues until only two players remain, who play the final round, that is, the last game, which determines the winner of the tournament.[^5]\n\nXIV Correspondence Internet Olympiad in Probability Theory and Statistics for Schoolchildren\n\n(c) Moscow Center for Continuous Mathematical Education, 2021\n\n26 participants arrived at the Anchurian Checkers tournament, and all of them play equally well, meaning that in any game played by any two of them, the chances of the opponents are equal. Among the players are Denis and Oleg. Find the probability that they will play against each other.",
        "answer": "\\frac{1}{13}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10743,
        "difficulty": null
      },
      {
        "id": "49c511f73c1d211f9f30ad46b6f6ff6cb3fbe6ce188d3c719f52a156d66463d7",
        "domain": "numina",
        "question": "[The ratio in which the bisector divides the side.] [Pythagorean Theorem (direct and inverse).]\n\nIn triangle $ABC$, the height $BM$, the bisector $BN$, and the median $BL$ are drawn. It is known that $AM = MN = NL$. Find the tangent of angle $A$ of this triangle.",
        "answer": "\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59828,
        "difficulty": null
      },
      {
        "id": "1a7863d961b40441b9301c13a49c5e224e3a4a8a1bb3a8d46c5c9d2a027d6b93",
        "domain": "numina",
        "question": "Problem 4.7. At the festival of namesakes, 45 Alexanders, 122 Borises, 27 Vasily, and several Gennady arrived. At the beginning of the festival, all of them stood in a row so that no two people with the same name stood next to each other. What is the minimum number of Gennady that could have arrived at the festival?",
        "answer": "49",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12082,
        "difficulty": null
      },
      {
        "id": "cdfbc5e238f66df74018c6879341c75b6bacdb95993b58a16a4825275275a58d",
        "domain": "numina",
        "question": "2. A cube with edge $a=1 / \\sqrt{2}$ is illuminated by a cylindrical beam of light with radius $\\rho=\\sqrt{2-\\sqrt{3}}$, directed along the main diagonal of the cube (the axis of the beam contains the main diagonal). Find the area of the illuminated part of the cube's surface.",
        "answer": "(\\sqrt{3}-\\frac{3}{2})(\\pi+3)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6348,
        "difficulty": null
      },
      {
        "id": "112a84f21ff4c55b3d17b9a8bf6b8df7d9e9d23d1ba6b8d2c3983085fadd7dab",
        "domain": "numina",
        "question": "20. Patrick asks SpongeBob: \"What is your lucky number?\" SpongeBob says: \"There is a sequence of numbers: $0,1,3,8$, $21,55,144,377,987, \\cdots \\cdots$ Starting from the third number, each number multiplied by 3 is exactly equal to the sum of the numbers immediately before and after it. My lucky number is the remainder when the 2020th number in this sequence is divided by 6.\" So, SpongeBob's lucky number is $\\qquad$ . (If it is divisible, then the remainder is 0)",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 92981,
        "difficulty": null
      },
      {
        "id": "b98b634202fea86e93a7fc1ec95e2616316ba3b379cd838344e9e7622959e5c0",
        "domain": "numina",
        "question": "Problem 2. Pentagon $A B C D E$ is inscribed in circle $\\omega$. Diagonal $A C$ is the diameter of circle $\\omega$. Find $\\angle B E C$, if $\\angle A D B=20^{\\circ}$.",
        "answer": "70",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12533,
        "difficulty": null
      },
      {
        "id": "c94fa19c41a42303a00efd465f3abfc13c5b14d5d24ab79bd144fa1c14ccdc5a",
        "domain": "numina",
        "question": "17. In a certain country, an aviation network has already been established, where any city is connected to no more than three other cities, and from any city to another city, it takes no more than one transfer. How many cities can this country have at most?",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 138435,
        "difficulty": null
      },
      {
        "id": "ec387db43cef8172f54808ad2d4e4aef999d6bcdc524e228225762c003bf6b0d",
        "domain": "numina",
        "question": "3 [Inscribed, circumscribed, and exscribed circles; their radii]\n\nTriangle $A B C$ is isosceles. The radius $O A$ of the circumscribed circle forms an angle $O A C$ with the base $A C$, which is $20^{\\circ}$. Find the angle $B A C$.",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64423,
        "difficulty": null
      },
      {
        "id": "fcbcb39109f1f17273aeaa73bb5a0ab1210a1cc77bca9186144502ff3f6ce4c7",
        "domain": "numina",
        "question": "Petr and Pavel were picking apples and pears in the orchard. On Monday, Petr ate 2 more pears than Pavel and 2 fewer apples than Pavel. On Tuesday, Petr ate 4 fewer pears than on Monday. Pavel ate 3 more pears than Petr and 3 fewer apples than Petr on Tuesday. Pavel ate 12 apples over the two days and the same number of apples as pears on Tuesday. In the evening on Tuesday, both boys found out that the number of apples they had eaten together over the two days was the same as the number of pears they had eaten together. How many apples did Petr eat on Monday and how many pears did Pavel eat on Tuesday?",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76733,
        "difficulty": null
      },
      {
        "id": "59ce77573cbe45e1ff6fabca5c567e8304bd29f25155fa543d79297a70faba66",
        "domain": "numina",
        "question": "## Task 1 - 160721\n\nAfter the coming-of-age ceremony, all students in a class had their individual photos taken. Each student had enough prints made of their photo, and then each student in this class exchanged their photo with each of their classmates.\n\nHow many students in this class exchanged photos with each other if exactly 812 photographs changed hands?",
        "answer": "29",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183659,
        "difficulty": null
      },
      {
        "id": "14ddac3258337bf6d9ad64d30483a5459b528669ccac6fd9603b9db3faa9fef8",
        "domain": "numina",
        "question": "3.48 Let a $10 \\times 10$ grid table $A$ contain an integer in each cell. Denote the sum of the numbers in the $i$-th row as $s_{i}$, and the sum of the numbers in the $j$-th column as $t_{j}, i, j=1,2, \\cdots, 10$. Then fill a new $10 \\times 10$ grid table $B$ in the following way: in the cell at the $i$-th row and $j$-th column, write the smaller of $s_{i}$ and $t_{j}$. It is known that the cells in table $B$ can be numbered from 1 to 100 such that the number in the $k$-th cell is no greater than $k$. Find the maximum possible sum of all the numbers in table $A$.",
        "answer": "955",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 131312,
        "difficulty": null
      },
      {
        "id": "a3c0671ff930eb737b4f1f7f16d1c2d2eb40ea55fe73420c60c952398c73c7c8",
        "domain": "numina",
        "question": "Bogdanov I.i.\n\nPetya wants to list all possible sequences of 100 natural numbers, in each of which a triple appears at least once, and any two adjacent members differ by no more than 1. How many sequences will he have to list?",
        "answer": "3^{100}-2^{100}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53758,
        "difficulty": null
      },
      {
        "id": "0ca9855d4997f0aee1ec22143ab6fd6fccfb21862a74acd137f190c15a9a8f8e",
        "domain": "numina",
        "question": "18 Real numbers $x, y, z$ not all positive satisfy $k\\left(x^{2}-x+1\\right)\\left(y^{2}-y+\\right.$ 1) $\\left(z^{2}-z+1\\right) \\geqslant(x y z)^{2}-x y z+1$, find the minimum value of the real number $k$.",
        "answer": "\\frac{16}{9}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 144648,
        "difficulty": null
      },
      {
        "id": "d005c75f845e66dadd0064fa0f925d4e9318d0fc9904cd9742496547866423f5",
        "domain": "numina",
        "question": "Problem 6. Having defeated Koschei, Ivan demanded gold to ransom Vasilisa from the bandits. Koschei led him to a cave and said:\n\n\"In the chest lie gold ingots. But you cannot simply take them: they are enchanted. Put one or several into your bag. Then I will put one or several from the bag back into the chest, but it must be a different number. We will take turns moving them: you into the bag, I into the chest, each time a new number. When a new move becomes impossible, you can take your bag with the ingots.\"\n\nWhat is the maximum number of ingots Ivan can take, no matter how Koschei acts, if the chest initially contains a) 13; b) 14 gold ingots? How can he do it?\n\n[8 points for both parts, 5 points for one] (A. V. Shapovalov)",
        "answer": "13",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22823,
        "difficulty": null
      },
      {
        "id": "88905094ef5b163fba2c8a4f05396ff9915eca5feb71d7591977775dc3b25a8a",
        "domain": "numina",
        "question": "7. If $x, y, k$ are positive reals such that\n$$\n3=k^{2}\\left(\\frac{x^{2}}{y^{2}}+\\frac{y^{2}}{x^{2}}\\right)+k\\left(\\frac{x}{y}+\\frac{y}{x}\\right) \\text {, }\n$$\nfind the maximum possible value of $k$.",
        "answer": "\\frac{-1+\\sqrt{7}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152234,
        "difficulty": null
      },
      {
        "id": "d46edd5e54409dd40d1903b1c38f970584761e7807029f836174d596eada9354",
        "domain": "numina",
        "question": "SI. 1 Given that $\\frac{3}{a}+\\frac{1}{u}=\\frac{7}{2}$ and $\\frac{2}{a}-\\frac{3}{u}=6$ are simultaneous equations in $a$ and $u$. Solve for $a$.",
        "answer": "\\frac{2}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152820,
        "difficulty": null
      },
      {
        "id": "ab1ad9faf3ad9ddfe45aec6572ce654a3c983a47dbf3da748aa0a2d9e4e32d2d",
        "domain": "numina",
        "question": "2. Add 3 digits after 325 to form a six-digit number, so that it can be divisible by $3$, $4$, and $5$, and make this value as small as possible. Then this new six-digit number is $\\qquad$ _.",
        "answer": "325020",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 128088,
        "difficulty": null
      },
      {
        "id": "b8fc898793446ad9760c194a21017096389fadc49be9d2f3baa1340b65eab487",
        "domain": "numina",
        "question": "$\\left[\\begin{array}{l}{[\\text { Transfer of sides, diagonals, etc. ] }} \\\\ {[\\text { Area of a trapezoid }}\\end{array}\\right]$\n\nFind the area of a trapezoid with bases 11 and 4 and diagonals 9 and 12.",
        "answer": "54",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67070,
        "difficulty": null
      },
      {
        "id": "f6d9cbec700a465be69f47d3386e9c252e11e2b542a9f856e3288601db32cdc4",
        "domain": "numina",
        "question": "3. Initially, the numbers $1, 2, 3, \\cdots, 2013$ are written on the blackboard. In each step, you can erase two numbers $a, b$ from the blackboard and write down $a+b$. What is the minimum number of steps required to ensure that the sum of any remaining numbers on the blackboard is not equal to 2014?\n(Lin Chang provided the problem)",
        "answer": "503",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 128186,
        "difficulty": null
      },
      {
        "id": "c8b0117ec4cb602c3608a6a93d5fba4f64c99f0bcb970ab866c2853c73c37260",
        "domain": "numina",
        "question": "1. (10 points) $\\frac{1-\\frac{1}{3}}{1 \\times \\frac{1}{2} \\times \\frac{1}{3}}+\\frac{\\frac{1}{3}-\\frac{1}{5}}{\\frac{1}{3} \\times \\frac{1}{4} \\times \\frac{1}{5}}+\\cdots+\\frac{\\frac{1}{2015}-\\frac{1}{2017}}{\\frac{1}{2015} \\times \\frac{1}{2016} \\times \\frac{1}{2017}}=$ $\\qquad$",
        "answer": "2034144",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111676,
        "difficulty": null
      },
      {
        "id": "fa42035cff587d1fb328059ab2cbe3a4ea75572bb07424c1e1394824614ec990",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\frac{e^{2 x}-e^{3 x}}{\\operatorname{arctg} x-x^{2}}$",
        "answer": "-1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47102,
        "difficulty": null
      },
      {
        "id": "144a204912ab9e8f8dccfa7cf7dc9c540d4ae4c2e1e5d717a24bbfce46d1070e",
        "domain": "numina",
        "question": "4. All natural numbers, the sum of the digits in the representation of which is divisible by 5, are listed in ascending order: $5,14,19,23,28,32, \\ldots$ What is the smallest positive difference between consecutive numbers in this sequence? Provide an example and explain why it cannot be smaller.\n\n---\n\nThe smallest positive difference between consecutive numbers in this sequence is 1. An example is the pair of numbers 14 and 15. The sum of the digits of 14 is $1 + 4 = 5$, which is divisible by 5. The sum of the digits of 15 is $1 + 5 = 6$, which is not divisible by 5. However, the next number in the sequence is 19, and the sum of its digits is $1 + 9 = 10$, which is divisible by 5. Therefore, the smallest difference is 1, as seen between 14 and 15.\n\nTo explain why it cannot be smaller: The smallest possible difference between two numbers is 1. If the difference were 0, the numbers would be the same, which is not possible for consecutive numbers in the sequence. If the difference were greater than 1, there would be a gap between the numbers, which means there would be a number in between that could potentially have a sum of digits divisible by 5, contradicting the requirement that the sequence lists all such numbers in ascending order. Therefore, the smallest possible difference is 1.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16136,
        "difficulty": null
      },
      {
        "id": "f61bbbf74e7f6b610297a2031e415b7656d1899d7cc0f925d3d527a6770282c9",
        "domain": "numina",
        "question": "5.3. For the celebration of the Name Day in the 5th grade parallel, several pizzas were ordered. 22 pizzas were ordered for all the boys, with each boy getting an equal share. Each girl also received an equal share, but half as much as each boy. How many pizzas were ordered if it is known that there are 13 girls in this parallel, and there are more boys than girls? Pizzas can be divided into parts.",
        "answer": "23",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16532,
        "difficulty": null
      },
      {
        "id": "7b8f05d0137c8c9f15be64d5a5805e3eb168f473de92d38f9563f4f32717d293",
        "domain": "numina",
        "question": "4. Suppose 40 objects are placed along a circle at equal distances. In how many ways can 3 objects be chosen from among them so that no two of the three chosen objects are adjacent nor diametrically opposite?",
        "answer": "7720",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 180417,
        "difficulty": null
      },
      {
        "id": "3d75535a9dcea23e563f7d186d35e54b33955d564cc58a2df704455c387f9a5b",
        "domain": "numina",
        "question": "4.4.9 ** Let $x_{1}, x_{2}, \\cdots, x_{n}$ and $a_{1}, a_{2}, \\cdots, a_{n}(n>2)$ be two sets of arbitrary real numbers satisfying the conditions:\n(1) $x_{1}+x_{2}+\\cdots+x_{n}=0$;\n(2) $\\left|x_{1}\\right|+\\left|x_{2}\\right|+\\cdots+\\left|x_{n}\\right|=1$;\n(3) $a_{1} \\geqslant a_{2} \\geqslant \\cdots \\geqslant a_{n}$. To make the inequality $\\mid a_{1} x_{1}+a_{2} x_{2}+\\cdots+ a_{n} x_{n} \\mid \\leqslant A\\left(a_{1}-a_{n}\\right)$ hold, find the minimum value of the number $A$.",
        "answer": "\\frac{1}{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 117708,
        "difficulty": null
      },
      {
        "id": "f9b4d44a258d73a2c3ed2040056fd588a2816d41abc96f745d3356656d72051d",
        "domain": "numina",
        "question": "For example, if you throw a die, you have two chances to throw. You can choose to throw once or twice, but the number of the last throw will be the final result. Find the mathematical expectation of the points obtained by such a throw. If you have three chances, still taking the number of the last throw as the final result, find the mathematical expectation of the points obtained by such a throw.",
        "answer": "\\frac{14}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 98713,
        "difficulty": null
      },
      {
        "id": "02eeb120318f1a3d4b8c6ce4269efcf67dd4d0314d1ccf096b50ada69ea24142",
        "domain": "numina",
        "question": "10. (16 points) Figure 1 is a rhombus paper piece composed of 2 small equilateral triangles; Figure 2 is a fixed regular hexagonal board $A B C D E F$, which is made up of 24 equally sized small equilateral triangles. Now, 12 rhombus paper pieces are used to completely cover the hexagonal board, there are $\\qquad$ different covering methods.",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 115723,
        "difficulty": null
      },
      {
        "id": "2cea7067a5743680eeac6b849e7f86808a0f2f9ed7442afbd3aeb3dc02eb8de2",
        "domain": "numina",
        "question": "$2 \\cdot 56$ Find the smallest natural number, such that when its last digit is moved to the first position, the number is multiplied by 5.",
        "answer": "142857",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112636,
        "difficulty": null
      },
      {
        "id": "aff00b3e06a7086434873076a0ccec11638c515edb00667ca63576cae40a3122",
        "domain": "numina",
        "question": "15. (6 points) A motorcycle travels a 60-kilometer distance at an average speed of 20 kilometers per hour, and on the return trip, its average speed is 30 kilometers per hour. What is the average speed of the motorcycle for the entire round trip?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126761,
        "difficulty": null
      },
      {
        "id": "580c8db4b45b1b1dbf80b828fbc6b822c30f8fadc3391dc6c3512d706afeb53d",
        "domain": "numina",
        "question": "# 2. Solution:\n\n$\\sin ^{4} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=1 \\Leftrightarrow\\left(1-\\cos ^{2} 2019 x\\right)^{2}+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=1 \\Leftrightarrow$ $\\cos ^{4} 2019 x-2 \\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=0 \\Leftrightarrow$ $\\cos ^{2} 2019 x\\left(\\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x-2\\right)=0 \\Leftrightarrow$ 1) $\\cos 2019 x=0, x=\\frac{\\pi}{4038}+\\frac{\\pi n}{2019}, n \\in Z$ or 2) $\\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x=2 \\Leftrightarrow$ $\\left\\{\\begin{array}{c}\\cos ^{2} 2019 x=1, \\\\ \\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x=1,\\end{array} \\Leftrightarrow\\left\\{\\begin{array}{l}\\sin 2019 x=0, \\\\ \\cos 2022 x=1,\\end{array} \\Leftrightarrow\\left\\{\\begin{array}{l}x=\\frac{\\pi n}{2019}, n \\in Z, \\\\ x=\\frac{\\pi m}{1011}, m \\in Z,\\end{array}\\right.\\right.\\right.$ $\\frac{n}{2019}=\\frac{m}{1011}, n=\\frac{2019 m}{1011}=\\frac{673 m}{337}, n \\in Z, m \\in Z \\Rightarrow m=337 k, \\quad x=\\frac{\\pi k}{3}, k \\in Z$",
        "answer": "\\frac{3}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10046,
        "difficulty": null
      },
      {
        "id": "f92fd138cafd5a7e8ea0d4b28eeeadd23a9217f66b55409d4d5ddbad4d491e5f",
        "domain": "numina",
        "question": "## Task 4 - 250934\n\nThe eight numbers 1, $2, \\ldots, 8$ are to be distributed to the vertices of a cube in such a way that the following conditions are met:\n\n(1) Each vertex of the cube is assigned exactly one of the eight numbers, and each of these numbers must appear in the distribution.\n\n(2) If one adds the four numbers at the vertices of each face of the cube, the sum on all six faces is the same.\n\nAs many distributions of the eight numbers to the vertices should be compiled as possible, such that each of these distributions meets conditions (1), (2), and no two of these distributions are congruent to each other, i.e., can be transformed into each other by rotation or reflection.\n\nDetermine the number of distributions that appear in such a compilation!",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 184685,
        "difficulty": null
      },
      {
        "id": "d56d258efe2b6eea06eca426b18bda0ad767efebcb787280a8dc9d26e0cbb4d4",
        "domain": "numina",
        "question": "Problem 2. Let $a, b, c, d$ be natural numbers such that $a+b+c+d=2018$. Find the minimum value of the expression\n\n$$\nE=(a-b)^{2}+2(a-c)^{2}+3(a-d)^{2}+4(b-c)^{2}+5(b-d)^{2}+6(c-d)^{2}\n$$",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178231,
        "difficulty": null
      },
      {
        "id": "048e4190aaa3ada9ba729ff6c12ec3158f37ab8797964def5f693967df9e29c7",
        "domain": "numina",
        "question": "$A B C$ is an equilateral triangle $(A B=a)$. Around vertex $A$, we draw two arcs, one with radius $r$ beyond the extensions of sides $A B$ and $A C$, and the other with radius $a+r$ between the rays of angle $B A C$. Similarly, we proceed around vertices $B$ and $C$, and the six arcs enclose a so-called Reuleaux $\\mathbf{R}$ domain of width $a+2 r$. We rotate this domain within a square frame of side $a+2 r$. What segments of the frame's sides does $\\mathbf{R}$ rub against during this process: 11[^0]\n\n\n[^0]:    ${ }^{1}$ Readers should cut out the shapes from cardboard, or draw one on transparent paper and move it over the other.",
        "answer": "(\\sqrt{3}-1)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174992,
        "difficulty": null
      },
      {
        "id": "9498af4317877fa75a963291a4bdf011a774b47f8421e1b9f2fe48c3396ab569",
        "domain": "numina",
        "question": "Problem 9.2. Vanya thought of two natural numbers, the product of which equals 7200. What is the greatest value that the GCD of these numbers can take?",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16732,
        "difficulty": null
      },
      {
        "id": "1623fea2e78f42581148c1b94a4307bcb288142f9ada1a057ee283be3af25cca",
        "domain": "numina",
        "question": "# 9. Solution.\n\n1st method. An elementary outcome in the random experiment is a triplet of positions where the children in green caps stand. Consider the event $A$ \"all three green caps are together.\" This event is favorable in 9 elementary outcomes. The event $B$ \"two green caps are together, and the third is separate\" is favorable in 45 elementary outcomes (9 ways to choose two adjacent places, and the third place must be one of the 5 places not adjacent to the already chosen ones). The total number of ways to choose a triplet is $C_{9}^{3}=84$. Then the probability of the event \"some two green caps are together\" is $\\mathrm{P}(A)+\\mathrm{P}(B)=\\frac{54}{84}=\\frac{9}{14}$, and the desired probability is $1-\\mathrm{P}(A)-\\mathrm{P}(B)=\\frac{5}{14}$.\n\n2nd method. Number the children in green caps and form pairs \"green-red\" such that a red cap is placed to the right of each green cap. To the right of the first green cap, any of the six red caps can be placed, to the right of the second, one of the five remaining, and to the right of the third, one of the four remaining red caps. This results in three pairs \"green-red\" and three separate red caps. There are exactly $6 \\cdot 5 \\cdot 4=120$ such groupings. Arrange these pairs and separate caps (a total of six elements) in a circle. This can be done in 5! ways, and no two green caps will be adjacent, as a red cap is located to the right of each green cap.\n\nThe total number of ways to arrange 9 children in a circle is 8!.\n\nThus, the desired probability is $\\frac{120 \\cdot 5!}{8!}=\\frac{120}{8 \\cdot 7 \\cdot 6}=\\frac{5}{14}$.\n\n3rd method is similar to the 2nd method from the first variant. Other methods of solution are possible.",
        "answer": "\\frac{5}{14}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 11003,
        "difficulty": null
      },
      {
        "id": "07a60c0f8b3deff09c6d3cf9d26eccfdfec6ad935ebef8cde6319cd32959cc63",
        "domain": "numina",
        "question": "1. Calculate: $7.625-6 \\frac{1}{3}+5.75-1 \\frac{3}{8}=$",
        "answer": "5\\frac{2}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110567,
        "difficulty": null
      },
      {
        "id": "2852c5426f68dd51791fdd02deabf6cdfeb8f1f4bdfef3336d86adab7bcb31dc",
        "domain": "numina",
        "question": "2. Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy, $a_{1}=1, a_{n+1}=5 a_{n}+1(n=1,2, \\cdots)$, then $\\sum_{n=1}^{2018} a_{n}=$",
        "answer": "\\frac{5^{2019}}{16}-\\frac{8077}{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95254,
        "difficulty": null
      },
      {
        "id": "66a18f3c87d8818e85f7335ff49ad62947baf283ef4f54d61dcd6a06dd690b28",
        "domain": "numina",
        "question": "11.5. In Flower City, there live $99^{2}$ dwarfs. Some of the dwarfs are knights (always tell the truth), while the rest are liars (always lie). The houses in the city are located in the cells of a $99 \\times 99$ square (a total of $99^{2}$ houses, arranged in 99 vertical and 99 horizontal streets). Each house is inhabited by exactly one dwarf. The house number is denoted by a pair of numbers $(x ; y)$, where $1 \\leqslant x \\leqslant 99$ is the number of the vertical street (numbers increase from left to right), and $1 \\leqslant y \\leqslant 99$ is the number of the horizontal street (numbers increase from bottom to top). The Flower City distance between two houses with numbers $\\left(x_{1} ; y_{1}\\right)$ and $\\left(x_{2} ; y_{2}\\right)$ is the number $\\rho=\\left|x_{1}-x_{2}\\right|+\\left|y_{1}-y_{2}\\right|$. It is known that on each street, whether vertical or horizontal, there live at least $k$ knights. In addition, all dwarfs know which house Knight Knowitall lives in. You want to find his house, but you do not know what Knowitall looks like. You can approach any house and ask the dwarf living there: “What is the Flower City distance from your house to the house of Knowitall?”. For what smallest $k$ can you guarantee to find the house of Knowitall? (V. Novikov)",
        "answer": "75",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18278,
        "difficulty": null
      },
      {
        "id": "6f9fe47bd8b5dd9b7145d121b2e0d3241864f32830420bc7a9530ce8a5d15227",
        "domain": "numina",
        "question": "For what value of $a$ does the polynomial $P(x)=x^{1000}+a x^{2}+9$ divide by $x+1$?",
        "answer": "-10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69240,
        "difficulty": null
      },
      {
        "id": "38a0c344a8a6b658f8d5f94f8befa127115363598e9e58013d27e70bc92018cb",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the equations in polar coordinates.\n\n$$\n\\rho=8 \\cos \\varphi, 0 \\leq \\varphi \\leq \\frac{\\pi}{4}\n$$",
        "answer": "2\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47336,
        "difficulty": null
      },
      {
        "id": "8c5312ebc8b649ac0b6c101b2f8128d483b0041064b38bb5b96cf946b2e07434",
        "domain": "numina",
        "question": "## Problem Statement\n\nFind the cosine of the angle between vectors $\\overrightarrow{A B}$ and $\\overrightarrow{A C}$.\n\n$A(2, -8, -1), B(4, -6, 0), C(-2, -5, -1)$",
        "answer": "-\\frac{2}{15}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47077,
        "difficulty": null
      },
      {
        "id": "70a170844f5ac54b2fc9ec0b94258c2bac9fe354ec73a88818de33ae7cf2d106",
        "domain": "numina",
        "question": "5. (15 points) As shown in the figure, the side length of the square is 80 cm, $O$ is the center of the square, $A$ is the midpoint of $O B$, and within the square, a circle is drawn with $A$ as the center and $O A$ as the radius. Another circle is drawn with $B$ as the center and $O B$ as the radius, forming a special shape with one side of the square. Rotating this shape around point $O$ three times in a clockwise direction forms a windmill shape. What is the area of this windmill (shaded part) in square centimeters? (Take $\\pi$ as 3.14)",
        "answer": "912",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143928,
        "difficulty": null
      },
      {
        "id": "0824ce1c57791fbe241620e10ec7b1ad2d10c0f3647b0d2514fe0d224aa7b70c",
        "domain": "numina",
        "question": "Example 4.8 Find the number of positive integer solutions to the equation $x_{1}+2 x_{2}+4 x_{3}=21$.\n\n untranslated text remains the same as requested.",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 122076,
        "difficulty": null
      },
      {
        "id": "4531d983acccf31e8a8f266add9ff4ce24988a66623058d61806cac2c02bb7cc",
        "domain": "numina",
        "question": "257. Given a regular tetrahedron with edge $a$. Find the radius of a sphere centered at the center of the tetrahedron, such that the combined volume of the part of the tetrahedron outside the sphere and the part of the sphere outside the tetrahedron is minimized.",
        "answer": "\\frac{1}{3}\\sqrt{\\frac{2}{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29958,
        "difficulty": null
      },
      {
        "id": "bcf84f339f2e41fee231438c74969214aece7911e1cb58b64adfe9e7cd5aae71",
        "domain": "numina",
        "question": "4. In tetrahedron $ABCD$, $\\angle ADB = \\angle BDC = \\angle CDA = 60^{\\circ}$, the areas of $\\triangle ADB$, $\\triangle BDC$, and $\\triangle CDA$ are $\\frac{\\sqrt{3}}{2}$, $2$, and $1$, respectively. Then the volume of the tetrahedron is . $\\qquad$",
        "answer": "\\frac{2\\sqrt{6}}{9}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 115339,
        "difficulty": null
      },
      {
        "id": "62b25082100f40be2a303d18119727673996295d4193ce39dd23bd9c8414aa21",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the areas of figures bounded by lines given in polar coordinates.\n\n$$\nr=\\cos \\phi, r=\\sqrt{2} \\sin \\left(\\phi-\\frac{\\pi}{4}\\right),\\left(-\\frac{\\pi}{4} \\leq \\phi \\leq \\frac{\\pi}{2}\\right)\n$$",
        "answer": "\\frac{\\pi}{4}-\\frac{1}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 45749,
        "difficulty": null
      },
      {
        "id": "edd5040bc7c58f051ce85f01751e1296f8c5e8d9fddf365fef9c4060502ec923",
        "domain": "numina",
        "question": "Kluci found an old minefield plan, see the picture. The numbers are on the fields where there are no mines, and they indicate the number of adjacent mined fields. Determine how many mines there are in the field and where they are. (Fields are adjacent if they share a vertex or a side.)\n\n| 1 |  | 2 |  | 2 |\n| :--- | :--- | :--- | :--- | :--- |\n|  | 3 |  | 3 |  |\n| 3 |  |  |  | 3 |\n|  | 2 |  |  |  |\n|  |  |  | 2 |  |",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76742,
        "difficulty": null
      },
      {
        "id": "8c54f408f0175d8c05e2410d8718af3af33a09851bd9a99818c26d4a4080ab03",
        "domain": "numina",
        "question": "15. Let the base of the pyramid $M-ABCD$ be a square, and $MA=MD, MA \\perp AB$. If the area of $\\triangle AMD$ is 1, find the radius of the largest sphere that can fit inside this pyramid.",
        "answer": "\\sqrt{2}-1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 125653,
        "difficulty": null
      },
      {
        "id": "d0c5581aef11c55148d480ad33cf1ef835cdb943cbdc8c461b4064e2985bd601",
        "domain": "numina",
        "question": "Zlobin C.A.\n\nThe greatest common divisor of natural numbers $m$ and $n$ is 1. What is the greatest possible value of $\\gcd(m+2000 n, n+2000 m) ?$",
        "answer": "3999999",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69709,
        "difficulty": null
      },
      {
        "id": "b68e1b5484d86727381b4d0f2d3ab0cd52f56cbeeadc13cdf3edde7d291ed96d",
        "domain": "numina",
        "question": "Problem 9.2. In four classes of a school, there are more than 70 children, all of whom came to the grade meeting (no other children were present at the meeting).\n\nEach girl who came was asked: \"How many people from your class, including you, came to the meeting?\"\n\nEach boy who came was asked: \"How many boys from your class, including you, came to the meeting?\"\n\nAmong the answers, the numbers $7,9,10,12,15,16,19$ and 21 were encountered (all children answered correctly).\n\n(a) (1 point) How many children are there in the largest class of the grade?\n\n(b) (3 points) How many girls came to the grade meeting?",
        "answer": "33",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13216,
        "difficulty": null
      },
      {
        "id": "99dd4c3328ec19055fdff4c83a1ac90698dd9708af81f231dd3925f119f29a9b",
        "domain": "numina",
        "question": "75. If $x+y=\\sqrt{3 \\sqrt{5}-\\sqrt{2}}, x-y=\\sqrt{3 \\sqrt{2}-\\sqrt{5}}$, then $xy=$",
        "answer": "\\sqrt{5}-\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120726,
        "difficulty": null
      },
      {
        "id": "902d7abddf8f86b3c943ba678abbb6053543c01b6c94d50d598bdb7417cd0a1d",
        "domain": "numina",
        "question": "11. Given $n^{2}=n \\times n$, find the last digit of $1^{2}+2^{2}+3^{2}+\\ldots+2016^{2}+2017^{2}$",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147205,
        "difficulty": null
      },
      {
        "id": "2534f662840abe52e3eaba84d9c8c58f2115c37ef1bcbcb9ec3091a5e28cf94b",
        "domain": "numina",
        "question": "10.3. Three people are playing table tennis, with the player who loses a game giving way to the player who did not participate in it. In the end, it turned out that the first player played 10 games, the second - 21. How many games did the third player play?",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16663,
        "difficulty": null
      },
      {
        "id": "56cddaf2b3c7f00c40321fcf7dc829f1d74f01f253c604fa05da9244f522004e",
        "domain": "numina",
        "question": "Problem 4. In triangle $A B C$ with the ratio of sides $A B: A C=7: 2$, the bisector of angle $B A C$ intersects side $B C$ at point $L$. Find the length of segment $A L$, if the length of the vector $2 \\cdot \\overrightarrow{A B}+7 \\cdot \\overrightarrow{A C}$ is 2016.",
        "answer": "224",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9454,
        "difficulty": null
      },
      {
        "id": "0c5533a721add533411d93ea9c44ea7facaeba464cee4d3ff9dc76ea0e7204c0",
        "domain": "numina",
        "question": "Problem 11.5. The tangent lines to the circumcircle $k$ of an isosceles $\\triangle A B C$, $A C=B C$, at the points $B$ and $C$ meet at point $X$. If $A X$ meets $k$ at point $Y$, find the ratio $\\frac{A Y}{B Y}$.\n\nEmil Kolev",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 177718,
        "difficulty": null
      },
      {
        "id": "da0170dc8e67a87d1cdc6d4e9a74f5df643313a50a96341460dda74690c29773",
        "domain": "numina",
        "question": "(2) Let $X=\\{1,2, \\cdots, 2001\\}$. Find the smallest positive integer $m$, such that: for any $m$-element subset $W$ of $X$, there exist $u, v \\in W$ (where $u$ and $v$ can be the same), such that $u+v$ is a power of 2.",
        "answer": "999",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144922,
        "difficulty": null
      },
      {
        "id": "1b86b878f01794b0327c5a1d4c144ac946aa53a050e00016f604674382bfbb08",
        "domain": "numina",
        "question": "3. (17 points) Find the smallest natural number that is simultaneously twice a perfect square and three times a perfect cube.\n\n#",
        "answer": "648",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4336,
        "difficulty": null
      },
      {
        "id": "2c1ed564dce1aafb3718b09117f9760414a008160ec7f0d367a05522c3a4a14f",
        "domain": "numina",
        "question": "1. Vasya and Petya drew a five-pointed star each. All the angles at the vertices of Petya's star are acute, while Vasya's star has an obtuse angle. Each of them claims that the sum of the angles at the vertices of his star is greater. Who is right?\n![](https://cdn.mathpix.com/cropped/2024_05_06_ad0ddfff017bcb029f08g-1.jpg?height=328&width=758&top_left_y=1595&top_left_x=654)",
        "answer": "180",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15979,
        "difficulty": null
      },
      {
        "id": "be8894cf1a722f55764e0840881e24070a654d068d360a25eff3f24b2eb23d9d",
        "domain": "numina",
        "question": "The midpoint of each side of the parallelogram is connected to the ends of the opposite side. Find the area of the octagon formed by the intersections of the drawn segments, if the area of the parallelogram is 1.\n\n#",
        "answer": "\\frac{1}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63358,
        "difficulty": null
      },
      {
        "id": "9c8396a77bf15d87b66e15f3086a0538673700dfa8b7227e645007921e8b8af6",
        "domain": "numina",
        "question": "39. Yura borrowed a book from Lena for three days. On the first day, he read half of the book, on the second day, he read a third of the remaining pages, and on the third day, he read a number of pages equal to half the number of pages he read in the first two days. Did Yura manage to finish the book in three days?",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39529,
        "difficulty": null
      },
      {
        "id": "e3aae5e76c1a0fb6efd0a0570d428ddc868c745df356ff695d82ce78d7f1ee64",
        "domain": "numina",
        "question": "1. (8 points) Calculate: $2013+201+20-13+3=$",
        "answer": "2224",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 111540,
        "difficulty": null
      },
      {
        "id": "6a1bb9aafd20b6e3e52636dfb19a2b7015c195d1bf6df9a53956a82c27922195",
        "domain": "numina",
        "question": "7. (10 points) The difference in area between two squares is 2016 square centimeters. If the side lengths of such pairs of squares are both integer centimeters, then how many pairs of squares satisfy the above condition.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 131348,
        "difficulty": null
      },
      {
        "id": "5b50f72729d60b3d44f47132e52afeccc882d8327a3c02532638c3a49ac895b2",
        "domain": "numina",
        "question": "8. (6 points) The three-digit number $2 a 3$ plus 326 equals $5 b 9$. If $5 b 9$ is a multiple of 9, then $a+b=$\n\n untranslated portion:\n```\n(6 points) The three-digit number $2 a 3$ plus 326 equals $5 b 9$. If $5 b 9$ is a multiple of 9, then $a+b=$\n```",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 130363,
        "difficulty": null
      },
      {
        "id": "4d33ef199a508cae0dc8624feb3363f76f532a1259cefa56da8b5b23ffb946e9",
        "domain": "numina",
        "question": "Kanel-Belov A.Y.\n\nA cube with a side of 20 is divided into 8000 unit cubes, and a number is written in each cube. It is known that in each column of 20 cubes, parallel to the edge of the cube, the sum of the numbers is 1 (columns in all three directions are considered). In a certain cube, the number 10 is written. Through this cube, three layers $1 \\times 20 \\times 20$, parallel to the faces of the cube, pass. Find the sum of all numbers outside these layers.",
        "answer": "333",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53608,
        "difficulty": null
      },
      {
        "id": "b0605e66a7a8d37118a3f622be305dc58b4331106f69d50be3b8f3d0487f78f3",
        "domain": "numina",
        "question": "6. Given 4 spheres with a radius of 1, each pair of which are externally tangent, then the edge length of the circumscribed regular tetrahedron of the geometric body formed by these 4 spheres is",
        "answer": "2\\sqrt{6}+2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 110205,
        "difficulty": null
      },
      {
        "id": "3032999f9c374434df5e996885beb62e5c6fe97b26d19e1bc2b6893b5f93d8f4",
        "domain": "numina",
        "question": "9. Solution. To get a complete collection, it is necessary that among the 9 coupons from Semy, there are 6 that Temy is missing. The probability of this is\n\n$$\n\\frac{C_{6}^{6} \\cdot C_{11}^{3}}{C_{17}^{9}}=\\frac{3}{442} \\approx 0.007\n$$",
        "answer": "0.007",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10706,
        "difficulty": null
      },
      {
        "id": "856c61f85db30bafefdd01e086747d1427e4bf870cbadc97016c48c9c4208a9a",
        "domain": "numina",
        "question": "33. How many six-digit numbers are there in which all digits are odd?",
        "answer": "15625",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 39302,
        "difficulty": null
      },
      {
        "id": "38e8f7bb9d5e16e3a3fba73ffe1e5ad93cbbbd257a7ef467629544ff52f2e3d3",
        "domain": "numina",
        "question": "$10 \\cdot 23$ has a four-digit number with thousands, hundreds, tens, and units digits being $a, b, c, d$ respectively, the sum of the digits is $26$, the tens digit of the product of $b, d$ equals $(a+c)$, and $(bd - c^2)$ is a power of 2. Find this four-digit number (explain the reasoning).\n(Heilongjiang Province, China Mathematics Competition, 1979)",
        "answer": "1979",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 104790,
        "difficulty": null
      },
      {
        "id": "57eebddc37747fef95fee015bde3287572008b61985138d2fe8a6b2d47ac59a9",
        "domain": "numina",
        "question": "XXII OM - III - Task 3\n\nHow many locks at least need to be placed on the treasury so that with a certain distribution of keys among the 11-member committee authorized to open the treasury, any 6 members can open it, but no 5 can? Determine the distribution of keys among the committee members with the minimum number of locks.",
        "answer": "462",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 1844,
        "difficulty": null
      },
      {
        "id": "9aef84255afbbdf08cc28f26b2221813d656d523ae99def492b9f737fdc84dd8",
        "domain": "numina",
        "question": "9. (12 points) Fill in the blanks with the numbers $2$, $0$, $1$, $5$, or leave them blank, so that each row and each column contains one $2$, $0$, $1$, $5$. The same number cannot be diagonally adjacent. Therefore, the first five positions of the fifth row are $\\qquad$ (use 9 to represent a blank).\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "15992",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 107569,
        "difficulty": null
      },
      {
        "id": "eb764d7fb93bd003b71ad2cb81195baa4d622853c301ed97481f9b3facadb1ce",
        "domain": "numina",
        "question": "Florinda went for a walk. She left point $A$ and walked $1 \\mathrm{~m}$. At this point, she turned left at an angle of $90^{\\circ}$ and walked $2 \\mathrm{~m}$. At the last point, she turned left again and walked $3 \\mathrm{~m}$. She continued walking in this manner until, on the last segment, she walked $30 \\mathrm{~m}$ and reached her final point, which we will call point $B$. The following figure illustrates the first seven segments of Florinda's walk (note that point $B$ is not illustrated).\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_463eeeabc25d438954b1g-13.jpg?height=523&width=536&top_left_y=1470&top_left_x=837)\n\nCalculate the distance between points $A$ and $B$.\n\n#",
        "answer": "\\sqrt{481}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73984,
        "difficulty": null
      },
      {
        "id": "934890148f8b832dc7b976c9f1dbcbe3076514e26179e0e79ebd7547bb5eccae",
        "domain": "numina",
        "question": "(15) Let $\\left\\{a_{n}\\right\\}$ be a geometric sequence, and each term is greater than 1, then $\\lg a_{1} \\lg a_{2012} \\sum_{i=1}^{20111} \\frac{1}{\\lg a_{i} \\lg a_{i+1}}$ has the value $\\qquad$.",
        "answer": "2011",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123631,
        "difficulty": null
      },
      {
        "id": "7c0022e452d0620e00f2bc164c0882cf7f3383c3ed2b75b298ab27e8e991d0fd",
        "domain": "numina",
        "question": "A number was written on the board. At each step, we add to it the largest of its digits (for example, if we have written 142, the next number will be 146). What is the maximum number of odd numbers that can be written consecutively by proceeding in this way?\n\n## High school statements",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78939,
        "difficulty": null
      },
      {
        "id": "c9f3f87effd4a142f68dd6602ed703aab34f0a8dd82fdbb013edf9190990d1bc",
        "domain": "numina",
        "question": "8. (10 points) Two small rulers form a set of rulers, and the small rulers can slide along the large ruler. Each unit on the large ruler is marked with a natural number. The first small ruler divides 11 units on the large ruler into 10, and the second small ruler divides 9 units on the large ruler into 10. The starting points of both small rulers are 0, and they are both marked from 1 to 10. Now, measure the distance between points A and B, where point A is at the 0 unit on the large ruler, and point B is between the 18th and 19th units on the large ruler. When the 0 unit of the first small ruler is placed at point B, its unit 3 coincides exactly with a unit on the large ruler. If the 0 unit of the second small ruler is placed at point B, then the \\_\\_\\_\\_\\_ unit of the second small ruler coincides exactly with a unit on the large ruler.",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 115604,
        "difficulty": null
      },
      {
        "id": "a8f02c8b24ec7ea35e3362adc9232cc354c009ca0f75228a9cc7d67d9fa89db8",
        "domain": "numina",
        "question": "The integers 390 and 9450 have three common positive divisors that are prime numbers. What is the sum of these prime numbers?",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194068,
        "difficulty": null
      },
      {
        "id": "0676db0c42751f7501402cb00d34c59cca686a17f5d69d3b836afdbeebe295e5",
        "domain": "numina",
        "question": "11th USAMO 1982 Problem 1 A graph has 1982 points. Given any four points, there is at least one joined to the other three. What is the smallest number of points which are joined to 1981 points? Solution",
        "answer": "1979",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 150105,
        "difficulty": null
      },
      {
        "id": "33f3b04bcfd575827fdac882294d86b0f7765faf2cc7dda2464a4e5f669f2170",
        "domain": "numina",
        "question": "## 226. Math Puzzle $3 / 84$\n\nSabine brings back 30 empty bottles, for which she receives $0.30 \\mathrm{M}$ refund per bottle.\n\nHow many bottles of soda costing $0.30 \\mathrm{M}$ each - for which she of course has to pay a deposit as well - could she buy again, and how much money, if any, would she have left over?",
        "answer": "29",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181897,
        "difficulty": null
      },
      {
        "id": "c41a4ac969ec8c71b8e0a1a2d83f16173235ffb6ca463656fe8b3d65c7db4ce8",
        "domain": "numina",
        "question": "7. Let $a, b, c$ be three distinct real numbers such that the equations $x^{2}+a x+1=0$ and $x^{2}+b x+c=0$ have a common real root, and the equations $x^{2}+x+a=0$ and $x^{2}+c x+b=0$ also have a common real root, then the value of $a+b+c$ is $\\qquad$.",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131625,
        "difficulty": null
      },
      {
        "id": "3596d39d9a448e04ef462e8cfe6c8e6c952e16dc8c12defdd811ea945363f35b",
        "domain": "numina",
        "question": "Example 8. Find the Wronskian determinant for the functions: $y_{1}(x)=\\sin x$,\n\n$$\ny_{2}(x)=\\sin \\left(x+\\frac{\\pi}{8}\\right), y_{3}(x)=\\sin \\left(x-\\frac{\\pi}{8}\\right)\n$$",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 34470,
        "difficulty": null
      },
      {
        "id": "f341a659559e9cb4d18533d68d4bb4301a30b6c6f89a27663bbe18d5a4aa6b7d",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=2 p+3 q$\n\n$b=p-2 q$\n\n$|p|=6$\n\n$|q|=7$\n\n$(\\widehat{p, q})=\\frac{\\pi}{3}$",
        "answer": "147\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 46987,
        "difficulty": null
      },
      {
        "id": "70bf1bc9ad1715c9f2844debc06010d5de71160884cd2d5093f307104f2a51f5",
        "domain": "numina",
        "question": "A3. We call a positive integer alternating if the digits of the number alternate between even and odd. Thus, 2381 and 3218 are alternating, but 2318 is not. We call a number completely alternating if the number itself is alternating and double the number is also alternating. Thus, 505 is completely alternating, because both 505 and 1010 are alternating numbers.\n\nHow many completely alternating four-digit numbers are there?\n\nNote: a four-digit number may not start with a 0.",
        "answer": "70",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86044,
        "difficulty": null
      },
      {
        "id": "c2b5917ec1d8f61c0ebe4fe8412bd28b84ec23e1b05be1f290184f2384901e1d",
        "domain": "numina",
        "question": "Kondakov G.V.\n\nDivide the segment $[-1,1]$ into black and white segments so that the integrals of any a) linear function;\n\nb) quadratic trinomial over the white and black segments are equal.\n\n#",
        "answer": "6",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 55351,
        "difficulty": null
      },
      {
        "id": "96dc1562ad19d665e79cf69b03bfddc502192e54dd466f1decad9b691391faaf",
        "domain": "numina",
        "question": "8. Specify the greatest value of the parameter $p$ for which the equation has at least one solution\n\n$2 \\cos \\left(2 \\pi-\\frac{\\pi x^{2}}{6}\\right) \\cos \\left(\\frac{\\pi}{3} \\sqrt{9-x^{2}}\\right)-3=p-2 \\sin \\left(-\\frac{\\pi x^{2}}{6}\\right) \\cos \\left(\\frac{\\pi}{3} \\sqrt{9-x^{2}}\\right)$. (16 points)",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10676,
        "difficulty": null
      },
      {
        "id": "045141012e43ae09ff85148555ad54d3b2c9dc8a998407932ac063ab9ebc887d",
        "domain": "numina",
        "question": "G2.2 Let $b=89+899+8999+89999+899999$, find the value of $b$.",
        "answer": "999985",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153050,
        "difficulty": null
      },
      {
        "id": "7e995ab759201e069b355f1ada4507da3126006445b8691bd45e8ee98993658b",
        "domain": "numina",
        "question": "5. Write down the 0 times, 1 times, 2 times, 3 times, 4 times, 5 times, ... of the natural number 15 in order: $0, 15, 30, 45, 60, 75, \\ldots$\n\nThis sequence of numbers can go on indefinitely, and each subsequent number is always larger than the previous one. Any natural number is either one of these numbers or lies between two adjacent numbers. We call this sequence a strictly increasing infinite sequence. Each number from left to right is called the first term, the second term, the third term, ..., i.e., the first term is 0, the second term is 15, the third term is $30, \\ldots$, and so on. Therefore, the number that lies between the 135th term and the 136th term of this sequence, and whose difference with the smaller of these two terms is 6, is",
        "answer": "2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145296,
        "difficulty": null
      },
      {
        "id": "78e686f23cfa36f89b5662ee8a546daeac2d67f2a52aaf9313c2e9e751d2ad74",
        "domain": "numina",
        "question": "6. Stopped Clock (from 6th grade. 1 point). On the clock, there are two hands: the hour hand and the minute hand. At a random moment in time, the clock stopped. Find the probability that the angle between the hands on the stopped clock is acute.",
        "answer": "0.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11053,
        "difficulty": null
      },
      {
        "id": "7810d4672871a11b3a701e95a2772a4345329f5cd59bc2b4ac48a87fcb689871",
        "domain": "numina",
        "question": "## Task B-2.3.\n\nThe lengths of the sides of the hexagon $A B C D E F$ are in order $|A B|=b,|B C|=|C D|=a,|D E|=b$, $|E F|=|F A|=a, a \\neq b$. A circle with radius 8 and center at point $O$ is circumscribed around the hexagon. If $\\cos \\varangle E F O=\\frac{\\sqrt{2}}{4}$, calculate the perimeter and area of this hexagon.",
        "answer": "16\\sqrt{2}+24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88766,
        "difficulty": null
      },
      {
        "id": "b262e78381f151f4ee1bfc690ee43e673571eb5aaf28042f9d72406f37baa5e9",
        "domain": "numina",
        "question": "5. Find the area of the cross-section of a regular hexagonal pyramid $S A B C D E F$ by a plane passing through the vertex $F$ of the base $A B C D E F$ and parallel to the median $C M$ of the lateral face $S C D$ and the apothem $S N$ of the lateral face $S A F$, if the side of the base of the pyramid is $4 \\sqrt{7}$, and the distance from the vertex $S$ to the cutting plane is $\\sqrt{7}$. (20 points)",
        "answer": "\\frac{202\\sqrt{3}}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10453,
        "difficulty": null
      },
      {
        "id": "344af0f950fce1b80a05e7f4e6ddc4108a11a0907e4adb02ae448e96bf3b7c50",
        "domain": "numina",
        "question": "Problem 9. The function $f$ is defined on the set of triples of integers and takes real values. It is known that for any four integers $a, b, c$ and $n$ the equalities $f(n a, n b, n c)=n \\cdot f(a, b, c)$, $f(a+n, b+n, c+n)=f(a, b, c)+n$, $f(a, b, c)=f(c, b, a)$ hold. Find $f(24,25,26)$.",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9403,
        "difficulty": null
      },
      {
        "id": "8a9f727a5ba55b20458e3415b654240abd05e19d419e6af94f2c72397a944117",
        "domain": "numina",
        "question": "G3.3 若方程 $\\sqrt[3]{5+\\sqrt{x}}+\\sqrt[3]{5-\\sqrt{x}}=1$, 求實數根 $x$ 。 Solve $\\sqrt[3]{5+\\sqrt{x}}+\\sqrt[3]{5-\\sqrt{x}}=1$ for real number $x$.",
        "answer": "52",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153423,
        "difficulty": null
      },
      {
        "id": "8b1079f4a8d00aeb698fa528d75c785c5bda596ba62f3d8ff1290f206abff0f0",
        "domain": "numina",
        "question": "9. Let $1<b<\\frac{5}{4}$, the number of distinct real roots of the equation $\\left|x^{2}-1\\right|=x+b$ is\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\n9. Let $1<b<\\frac{5}{4}$, the number of distinct real roots of the equation $\\left|x^{2}-1\\right|=x+b$ is",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129188,
        "difficulty": null
      },
      {
        "id": "015298f7cd136112c5bafa6866e2dfa3f9cceebe324e0f52f210d30a2e9f6690",
        "domain": "numina",
        "question": "(4) If $4n+1, 6n+1$ are both perfect squares, then the smallest positive integer value of $n$ is\n$\\qquad$",
        "answer": "20",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 135813,
        "difficulty": null
      },
      {
        "id": "9833af81ba3656af51ddaba3e0557a897c5d2e4c3cab681c57eb8cee719aecd8",
        "domain": "numina",
        "question": "4. Let $I$ be the incenter of $\\triangle A B C$, and $3 \\overrightarrow{I A}+4 \\overrightarrow{I B}+5 \\overrightarrow{I C}=0$, then the size of angle $C$ is",
        "answer": "\\frac{\\pi}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142654,
        "difficulty": null
      },
      {
        "id": "7a9647b815baeff65e1d714be4a0cd1326de392aba99e1d64fb67b3b5c664829",
        "domain": "numina",
        "question": "4. In a family, there are six children. Five of them are respectively 2, 6, 8, 12, and 14 years older than the youngest, and the age of each child is a prime number. How old is the youngest?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35628,
        "difficulty": null
      },
      {
        "id": "de726b5ae0c10e2466563beace0723ae6f00f2eeaee4d3c7462804799f9e9da1",
        "domain": "numina",
        "question": "Problem 3. (Option 2)\n\nFind the sum of the squares of the roots of the equation $\\left(x^{2}+6 x\\right)^{2}-1580\\left(x^{2}+6 x\\right)+1581=0$.",
        "answer": "3232",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10424,
        "difficulty": null
      },
      {
        "id": "fce4f0de95e00246753c32973fdd84b98f2632dfab173c6fe53446d2702f74aa",
        "domain": "numina",
        "question": "14. The function $f(x)$ defined on the interval $[1,2017]$ satisfies $f(1)=f(2017)$, and for any $x, y \\in [1,2017]$, it holds that $|f(x)-f(y)| \\leqslant 2|x-y|$. If the real number $m$ satisfies that for any $x, y \\in [1,2017]$, it holds that $|f(x)-f(y)| \\leqslant m$, find the minimum value of $m$.",
        "answer": "2016",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 136921,
        "difficulty": null
      },
      {
        "id": "1cb8b9ce415cfab6e807015e246d0dbd69eda0a3ff314c6eaeb3e48390253e0c",
        "domain": "numina",
        "question": "# Task № 5.4\n\n## Condition:\n\nA Dog, a Cat, and a Mouse are running around a circular lake. They started simultaneously in the same direction from the same point and finished simultaneously, all running at constant speeds.\n\nThe Dog ran 12 laps, the Cat ran 5 laps, and the Mouse ran 2 laps. How many total overtakes were made from the start to the finish?\n\nIf two or more overtakes occur simultaneously, each overtake is counted separately. The start and finish moments are not counted as overtakes.",
        "answer": "17",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13895,
        "difficulty": null
      },
      {
        "id": "81763f5809c5466ee195378b1e3987703a824fab66c3be831a55f314a4d1d328",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{\\pi} 2^{4} \\cdot \\cos ^{8}\\left(\\frac{x}{2}\\right) d x\n$$",
        "answer": "\\frac{35\\pi}{8}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46277,
        "difficulty": null
      },
      {
        "id": "c01fb39a1150649163cf604b7baf8e3eb13958268cc0a5f75c66b398c0413045",
        "domain": "numina",
        "question": "3. 11088 is $110\\%$ of the containers in December compared to November. This means that in November, $11088: 1.10=10080$ containers were manufactured, which is $105\\%$ compared to October. This means that in October, $10080: 1.05=9600$ containers were manufactured, which is $120\\%$ compared to September. Therefore, in September, $9600: 1.20=8000$ containers were manufactured.",
        "answer": "8000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12205,
        "difficulty": null
      },
      {
        "id": "3d28fc625e4755e178b20d5f897cee52af48fa4503a5f315c292614510d51abe",
        "domain": "numina",
        "question": "A8. The sum of the numbers 1 to 123 is 7626 . One number is omitted so that the sum is now a multiple of 4 . How many different numbers could be omitted?",
        "answer": "31",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 155990,
        "difficulty": null
      },
      {
        "id": "7b337a9ba931299d89a09988619c64b95d4c81f82b6680cd748f7fa39a89b81d",
        "domain": "numina",
        "question": "Problem 11.2. In the store, there are 9 headphones, 13 computer mice, and 5 keyboards. In addition, there are 4 sets of \"keyboard and mouse\" and 5 sets of \"headphones and mouse\". In how many ways can you buy three items: headphones, a keyboard, and a mouse? Answer: 646.",
        "answer": "646",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12133,
        "difficulty": null
      },
      {
        "id": "9d91384d1b6a85759fc46ca6592c5c49a302b50d87f071591211618199b671db",
        "domain": "numina",
        "question": "\\section*{Exercise 1 - 281021}\n\nThe smallest positive natural number is sought, whose digit representation (in the decimal system) consists only of the digits 0 and 1 and which is divisible by 450.",
        "answer": "11111111100",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 185372,
        "difficulty": null
      },
      {
        "id": "7b7b9080153d0784ebc5cf0ff1850555070fbd92e3683a62f1c4b2ed417d8715",
        "domain": "numina",
        "question": "4 [ Excircle ]\n\nLet $A B C$ be an equilateral triangle. Three planes through the lines $A B, B C$, and $A C$ form an angle $\\phi$ with the plane $A B C$ and intersect at point $D 1$. In addition, three planes through the same lines form an angle $2 \\phi$ with the plane $A B C$ and intersect at point $D 2$. Find $\\phi$, given that points $D 1$ and $D 2$ are at equal distances from the plane $A B C$.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62863,
        "difficulty": null
      },
      {
        "id": "079ba5675a5806ea65508906f4aad78c2bb45167c064198e279dd75974376a81",
        "domain": "numina",
        "question": "7.98 Two people are playing a math game. Player A selects a set of 1-digit integers $x_{1}, x_{2}, \\cdots, x_{n}$ as the answer, which can be positive or negative. Player B can ask: what is the sum $a_{1} x_{1}+a_{2} x_{2}+\\cdots+a_{n} x_{n}$? Here, $a_{1}, a_{2}, \\cdots, a_{n}$ can be any set of numbers. Determine the minimum number of questions Player B needs to ask to guess the answer:",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110865,
        "difficulty": null
      },
      {
        "id": "f77757bc0e74d2aff7bc352d9b8e2c1dc8decd7fe12ada45e26242307bf81e7a",
        "domain": "numina",
        "question": "Problem 10-5. Consider all reduced quadratic trinomials $x^{2}+p x+$ $q$ with integer coefficients $p$ and $q$. Let's call the range of such a trinomial the set of its values at all integer points $x=0, \\pm 1, \\pm 2, \\ldots$ What is the maximum number of such trinomials that can be chosen so that their ranges do not intersect pairwise?\n\nAnswer: 2.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 20126,
        "difficulty": null
      },
      {
        "id": "575f555ee61f1d450bdd897465ee0ce887cd38e0488f438482583b84ac27cc72",
        "domain": "numina",
        "question": "10. Determine the value of\n$$\n2002+\\frac{1}{2}\\left(2001+\\frac{1}{2}\\left(2000+\\cdots+\\frac{1}{2}\\left(3+\\frac{1}{2} \\cdot 2\\right)\\right) \\cdots\\right)\n$$",
        "answer": "4002",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152121,
        "difficulty": null
      },
      {
        "id": "480bc57812d9e9d3ffea231a037bf99f9e15a3ac0d5d29cdecb06a191b58771d",
        "domain": "numina",
        "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhat is greater: $7^{92}$ or $8^{91}$ ?",
        "answer": "8^{91}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 68383,
        "difficulty": null
      },
      {
        "id": "b6132f42d678795886a660787a5ca18211ac4bc011761c4ea956a9f77370d08b",
        "domain": "numina",
        "question": "I3.4 If -2 and $216 c$ are the roots of the equation $p x^{2}+d x=1$, find the value of $d$.",
        "answer": "-\\frac{7}{18}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153030,
        "difficulty": null
      },
      {
        "id": "c0680d2a7a948c8df9c64ac33427c66a65463419156a4ec8e9f2a97c9c541409",
        "domain": "numina",
        "question": "A3. What is the largest integer for which each pair of consecutive digits is a square?",
        "answer": "81649",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 155937,
        "difficulty": null
      },
      {
        "id": "04986de4e5078f7f9d9ff4abf3735bc686be6fa93b220a443e2e1f6e5873dbcc",
        "domain": "numina",
        "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 1, 12, and 12, and the vertex angles are $-4 \\operatorname{arctg} \\frac{1}{3}, 4 \\operatorname{arctg} \\frac{2}{3}$, and $4 \\operatorname{arctg} \\frac{2}{3}$ respectively (the vertex angle of a cone is the angle between its generators in the axial section). A sphere is placed on the table, touching all the cones. Find the radius of the sphere.",
        "answer": "\\frac{40}{21}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7652,
        "difficulty": null
      },
      {
        "id": "53bab686e927f89e3b162a6c4ee1b1e968d790f154cd5cc2d746d22b9549c503",
        "domain": "numina",
        "question": "B1. Mirko performs in the circus in the \"cannon launch\" act. His motion, which is shown in the figure, is described by the function $f$ with the rule\n\n$$\nf(x)=x-\\frac{1}{50} x^{2}\n$$\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_d6b556bd14f8001ca6fag-07.jpg?height=277&width=825&top_left_y=224&top_left_x=1092)\n\na) How far from the launch point should the center of the net be placed so that Mirko lands safely in it?\n\nb) What is the maximum height above the ground that Mirko reaches?",
        "answer": "50",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 164221,
        "difficulty": null
      },
      {
        "id": "841de4c5da224c8366fd76a6be656f84402ef86ae57c40907efeb577f014f6a5",
        "domain": "numina",
        "question": "Problem 2. In the class, two students sit at each desk. The number of desks where two boys sit is twice the number of desks where two girls sit. And the number of desks where two girls sit is twice the number of desks where a boy and a girl sit. How many boys are there in the class if there are 10 girls?\n\nAnswer: there are 18 boys in the class.",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14151,
        "difficulty": null
      },
      {
        "id": "d006b1b542dca5b7a68fd10e5cdf4efb1bdbf9852777acfbe39f54671bf1b5c0",
        "domain": "numina",
        "question": "8. Suppose there are 6 different elements of class A and 4 different elements of class B. An experiment requires an even number of elements from class A and at least 2 elements from class B. How many experimental schemes are there?\n\n保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "353",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 114674,
        "difficulty": null
      },
      {
        "id": "887d33ae60aaab64a5589bfdf734372626107a67ab35ad191efab83bc4f63791",
        "domain": "numina",
        "question": "## SUBJECT 3\n\nLet the number $\\mathrm{n}=9+99+999+\\ldots+\\underbrace{99 \\ldots 9}_{\\text {2016 digits }}+2016$.\n\na) Show that the number n is divisible by 90.\n\nb) Find the quotient and remainder of the division of $\\mathrm{n}$ by 11111.",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 179831,
        "difficulty": null
      },
      {
        "id": "553a7318ff6d6502b5847afbbdb128fef87ffae33994d202ab4bec9e422b5e5a",
        "domain": "numina",
        "question": "G2.3 Determine the sum of all real roots of the following equation $|x+3|-|x-1|=x+1$.",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153386,
        "difficulty": null
      },
      {
        "id": "4df7feee8f4401ac47af5111edbd5ad955b9a7a0380339d3ac7b2a5ef910f765",
        "domain": "numina",
        "question": "13. [9] Suppose $\\triangle A B C$ has lengths $A B=5, B C=8$, and $C A=7$, and let $\\omega$ be the circumcircle of $\\triangle A B C$. Let $X$ be the second intersection of the external angle bisector of $\\angle B$ with $\\omega$, and let $Y$ be the foot of the perpendicular from $X$ to $B C$. Find the length of $Y C$.",
        "answer": "\\frac{13}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151696,
        "difficulty": null
      },
      {
        "id": "9995f497c80489eb6853bba37f94723958e2fed37002726155bf43e5f65e4a19",
        "domain": "numina",
        "question": "11. (20 points) Let $x, y$ be complex numbers, find the maximum value of $\\frac{|3 x+4 y|}{\\sqrt{|x|^{2}+|y|^{2}+\\left|x^{2}+y^{2}\\right|}}$.",
        "answer": "\\frac{5\\sqrt{2}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140660,
        "difficulty": null
      },
      {
        "id": "fb4d3acea5975ea860744c3c4b58bbc045755df2311d8c32acca77202555de7d",
        "domain": "numina",
        "question": "## Task B-1.1.\n\nFor positive real numbers $a$ and $b$, define $a \\star b=\\frac{a-b}{a+b}$. What is\n\n$$\n\\frac{(a \\star b)+1}{(b \\star a)-1}+\\frac{(a \\star b)+2}{(b \\star a)-2}+\\ldots+\\frac{(a \\star b)+k}{(b \\star a)-k}+\\ldots+\\frac{(a \\star b)+2020}{(b \\star a)-2020}\n$$",
        "answer": "-2020",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89268,
        "difficulty": null
      },
      {
        "id": "c3e335cb0354c693da1919e1aed799b8f6e67a21c707a81ee5a13a096122a059",
        "domain": "numina",
        "question": "In the following figure, all the small squares on the board are equal. What percentage of the larger square does the painted region cover?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_e67c0440f1fa8bc62cdcg-21.jpg?height=219&width=217&top_left_y=1482&top_left_x=994)\n\n#",
        "answer": "18.75",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73338,
        "difficulty": null
      },
      {
        "id": "dc57afa64c129bdaea08565dbd2d180c7da82b8a4e0f13fd4bcf3f6e03e3392e",
        "domain": "numina",
        "question": "3.184. Given the fraction $\\frac{5}{1+\\sqrt[3]{32 \\cos ^{4} 15^{\\circ}-10-8 \\sqrt{3}}}$. Simplify the expression under the cube root, and then reduce the fraction.",
        "answer": "1-\\sqrt[3]{4}+\\sqrt[3]{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50643,
        "difficulty": null
      },
      {
        "id": "be68af7c97068152a6c915696e6bdfa069311313170fa60f514ac2d21460b58b",
        "domain": "numina",
        "question": "## Task 5 - 290735\n\nWe consider the product of all natural numbers from 1 to 1000 inclusive.\n\nDetermine the number of zeros with which this product ends!",
        "answer": "249",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 183821,
        "difficulty": null
      },
      {
        "id": "4f8ad1271bfb6bce4af71df78b0c9b4f31c21c03520d2d7aa7dc5d8ad8913c6e",
        "domain": "numina",
        "question": "## Task A-4.5.\n\nWhat is the probability that for two randomly chosen numbers $x, y$ from the set $[-2,2]$ the following holds\n\n$$\n|x|+|y| \\geqslant 1 \\quad \\text { and } \\quad \\| x|-| y|| \\leqslant 1 ?\n$$",
        "answer": "\\frac{5}{8}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 88831,
        "difficulty": null
      },
      {
        "id": "68f5879d3a4c61c4a0fb063f9c7b7fa88d6786713df6c3bc2f1769d89337401c",
        "domain": "numina",
        "question": "Folklore\n\nA narrow ribbon is wound around a cylindrical column that is 20 meters high and 3 meters in diameter, rising from the base to the top in seven full turns. What is the length of the ribbon?",
        "answer": "\\sqrt{400+441\\pi^{2}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64804,
        "difficulty": null
      },
      {
        "id": "db99c69546f3b68f1b692d6be5aa4f4bb291cb180268b816cdc543254a3d9705",
        "domain": "numina",
        "question": "The cities of Coco da Selva and Quixajuba are connected by a bus line. Buses leave Coco da Selva for Quixajuba every hour, with the first one departing at midnight on the dot. Buses leave Quixajuba for Coco da Selva every hour, with the first one departing at half past midnight. The bus journey takes exactly 5 hours.\n\nIf a bus leaves Coco da Selva at noon, how many buses coming from Quixajuba does it encounter during the journey?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=226&width=317&top_left_y=1643&top_left_x=1506)\n\nFigure 21.2\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=249&width=305&top_left_y=1943&top_left_x=1521)\n\nFigure 21.3\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=220&width=214&top_left_y=2260&top_left_x=1566)\n\nFigure 21.4",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 74035,
        "difficulty": null
      },
      {
        "id": "f2ae189102a4eacce47c46069da7e854ec6fc8caf005da163c43170ebd394cb2",
        "domain": "numina",
        "question": "5. For any $n \\in \\mathbf{N}^{*}$, the parabola $y=\\left(n^{2}+n\\right) x^{2}-(2 n+1) x+1$ intersects the $x$-axis at points $A_{n}$ and $B_{n}$. Then $\\left|A_{1} B_{1}\\right|+\\left|A_{2} B_{2}\\right|+\\cdots+\\left|A_{2009} B_{2009}\\right|=$ $\\qquad$",
        "answer": "\\frac{2009}{2010}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108906,
        "difficulty": null
      },
      {
        "id": "144538672d9b24d06ea5c5b72945d90b86f62592970c82f5c9b3c597e9e0e2f7",
        "domain": "numina",
        "question": "9.5. On the side $AC$ of triangle $ABC$, a point $M$ is taken. It turns out that $AM = BM + MC$ and $\\angle BMA = \\angle MBC + \\angle BAC$. Find $\\angle BMA$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22549,
        "difficulty": null
      },
      {
        "id": "e7329fe681c16a1e01479f455a429d074971cfba9a8538aa35869e0197c988c6",
        "domain": "numina",
        "question": "Let's find a point that is at distances $m, n, p$ from three planes. How many solutions does this problem have?",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174733,
        "difficulty": null
      },
      {
        "id": "02e8d001dbd0a5917d9fc0609f90d4d0189c105651d15a2aca2c388feb44a519",
        "domain": "numina",
        "question": "3. Coaches Ivan and Marko are organizing a treasure hunt through two forest trails. Katja walked both trails, Ivan's and Marko's, and took a total of 2024 steps of the same length. If she had taken 204 fewer steps on Ivan's trail and 24 more steps on Marko's trail, she would have taken the same number of steps on both trails. How many steps did Katja take on Ivan's trail, and how many on Marko's trail?",
        "answer": "898",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86690,
        "difficulty": null
      },
      {
        "id": "ee32effb422f5fc9925e311fb5fabeac2ff3dd6f787fcc56a681da30413f06b8",
        "domain": "numina",
        "question": "8. Arrange the numbers $1,2, \\cdots, 6$ in any order. If the number $k$ appears exactly in the $k$-th position, it is called a match. The expected number of matches is $\\qquad$",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 124220,
        "difficulty": null
      },
      {
        "id": "bcd56312dfcb0c368caa96cf990c9b4da4564865cbd9478054bb07e3fa009d3c",
        "domain": "numina",
        "question": "Task B-4.3. Determine all numbers $z \\in \\mathbb{C}$ that are solutions to the equation $z^{3}+i^{2013}=0$. Calculate the area of the figure whose vertices in the Gaussian plane are determined by the solutions to this equation.",
        "answer": "\\frac{3\\sqrt{3}}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89222,
        "difficulty": null
      },
      {
        "id": "42aee9a1dd9eaeb59fdf4d49a29e92b928b7d640ea66c83d116b51bd71481bf5",
        "domain": "numina",
        "question": "7.2. There are nuts in three boxes. In the first box, there are six nuts fewer than in the other two boxes combined, and in the second box, there are ten fewer nuts than in the other two boxes combined. How many nuts are in the third box? Justify your answer.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12825,
        "difficulty": null
      },
      {
        "id": "044fe1370d5b091b05ac164c2ab2220ff8e567129da5c2b07f6e74325ea6140d",
        "domain": "numina",
        "question": "6. Let's call the distance between numbers the absolute value of their difference. It is known that the sum of the distances from sixteen consecutive natural numbers to some number $a$ is 636, and the sum of the distances from these same sixteen numbers to the number $a^{2}$ is 591. Find all possible values of $a$.",
        "answer": "-\\frac{5}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3513,
        "difficulty": null
      },
      {
        "id": "efa321adc79ee80d19e45706dfee8eaac333c49f291be760e4a5ee7eeb903bbf",
        "domain": "numina",
        "question": "12. Two squares $A B C D$ and $B E F G$ with side lengths of $8 \\mathrm{~cm}$ and $6 \\mathrm{~cm}$ respectively are placed side by side as shown in the figure. Connecting $D E$ intersects $B G$ at $P$. What is the area of the shaded region $A P E G$ in the figure?",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124505,
        "difficulty": null
      },
      {
        "id": "3a5bcdeaf83472d46e62c4c4b2c64029ccee9ec678713deb3c9a17718e07abab",
        "domain": "numina",
        "question": "What are the last two digits of $9^{h}$, starting from the number $a=1$,\n\n$$\nb=2^{a}, \\quad c=3^{b}, \\quad d=4^{c}, \\quad e=5^{d}, \\quad f=6^{e}, \\quad g=7^{f}, \\quad h=8^{g} ?\n$$",
        "answer": "21",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 164984,
        "difficulty": null
      },
      {
        "id": "f357628b45c0f3d6f47b86e8e66082969c3209cf9289368f5c0b9d2354877127",
        "domain": "numina",
        "question": "13. $[8]$ Find the smallest positive integer $n$ such that $\\frac{5^{n+1}+2^{n+1}}{5^{n}+2^{n}}>4.99$.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151468,
        "difficulty": null
      },
      {
        "id": "60f51395591c988c206b20ba10c956a8ef3119d84f8d20331d1391b59ba17bab",
        "domain": "numina",
        "question": "10-2-1. Petya writes down a sequence of numbers: if the current number is equal to $x$, then the next one is $\\frac{1}{1-x}$. The first number in the sequence is 2. What is the five hundredth number?",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16986,
        "difficulty": null
      },
      {
        "id": "ca29dd641f0044c8b25cbddd8f2f734a53d65a0f4eaa2aab2f1aca2bfdc7d962",
        "domain": "numina",
        "question": "Task 10.4. In the central cell of a $21 \\times 21$ board, there is a chip. In one move, the chip can be moved to an adjacent cell by side. Alina made 10 moves. How many cells can the chip end up in?",
        "answer": "121",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12908,
        "difficulty": null
      },
      {
        "id": "f642cbc35f0ba7e0918d7c7c7c8aa6a17676f43b986747c314c9590baa0b8ba2",
        "domain": "numina",
        "question": "3. Given that the radius of the circumcircle $\\odot O$ of square $ABCD$ is 10. Draw a semicircle outside the square with $AB$ as the diameter. Find the area of the region of the semicircle that lies outside $\\odot O$.",
        "answer": "50",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144221,
        "difficulty": null
      },
      {
        "id": "9c7ae98a79a9f2102787ac111b1706425d82f947dc5f31bc1e5305cb6fa4e872",
        "domain": "numina",
        "question": "3 $[\\quad$ Case Analysis $\\quad]$\n\nGiven five different positive numbers that can be divided into two groups such that the sums of the numbers in these groups are equal. In how many ways can this be done?\n\n#",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53334,
        "difficulty": null
      },
      {
        "id": "a4fa79c19842864d9a60788d3a1f0ed9586e39178fa81c0cc216444d06afa942",
        "domain": "numina",
        "question": "4- 120 If $\\operatorname{tg} x+\\operatorname{tg} y=25$, and $\\operatorname{ctg} x+\\operatorname{ctg} y=30$. Find $\\operatorname{tg}(x+y)$.",
        "answer": "150",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136464,
        "difficulty": null
      },
      {
        "id": "bd73c72ac2ca47865e48ff814b198e86d88e897f2b684a53b6fa6872dd826007",
        "domain": "numina",
        "question": "Problem 3. Senya cannot write some letters and always makes mistakes in them. In the word TETRAHEDRON he would make five mistakes, in the word DODECAHEDRON - six, and in the word ICOSAHEDRON - seven. How many mistakes would he make in the word OCTAHEDRON?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5f8bb630dfa91fa0065fg-2.jpg?height=41&width=471&top_left_y=799&top_left_x=802)",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22845,
        "difficulty": null
      },
      {
        "id": "2e922a254e5f691a48219de464817b6d9235e1385c3562966c2e2de519019d83",
        "domain": "numina",
        "question": "3. Let $n$ be a natural number, for any real numbers $x, y, z$ there is always $\\left(x^{2}+y^{2}+z^{2}\\right) \\leqslant n\\left(x^{4}+y^{4}+z^{4}\\right)$, then the minimum value of $n$ is $\\qquad$",
        "answer": "3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 97473,
        "difficulty": null
      },
      {
        "id": "3861f8dbf9b4f35ed9955fee938c9bb33e3bcca6875c5ca3f95a60ba8ab4ebd0",
        "domain": "numina",
        "question": "4. $\\mathrm{ABCD}$ is a rhombus. Points $E$ and $F$ lie on sides $AB$ and $BC$ respectively, such that $\\frac{AE}{BE}=\\frac{BF}{CF}=5$. Triangle $DEF$ is equilateral. Find the angles of the rhombus. Solution.\n\nThe notation is shown in the figure. The situation where angle $D$ of the rhombus is acute is impossible (see below). Mark point $P$ on side $AB$ such that $AP=EB=FC$. Triangles $CDF$ and $ADP$ are congruent, as they have equal angles $\\angle A=\\angle C$ and corresponding equal sides $AD=DC, AP=FC$. Considering that triangle $FED$ is equilateral by the condition, we get that triangle $EDP$ is isosceles $(ED=FD=PD)$. (At this point, note that if angle $\\angle A$ of the rhombus is obtuse, then angle $\\angle APD$ is acute, and in the isosceles triangle $EDP$, the angle at the base is obtuse,\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_aa449fffebf3e1dc9820g-3.jpg?height=499&width=1037&top_left_y=116&top_left_x=521)\n\nwhich is impossible). Mark the equal angles $x=\\angle A=\\angle C$ and $y=\\angle ADP=\\angle CDF$. We find $\\angle EPD=x+y$ (the exterior angle of triangle $ADP$) and $\\angle PDE=180^{\\circ}-2(x+y)$. Summing all the angles at vertex $D$ of the rhombus ($\\left.\\angle D=180^{\\circ}-x\\right)$, we get the equation\n\n$$\ny+180^{\\circ}-2(x+y)+60^{\\circ}+y=180^{\\circ}-x\n$$\n\nFrom this, $x=60^{\\circ}$, the angles of the rhombus are $60^{\\circ}$ and $120^{\\circ}$.\n\nThe end of the solution can be different. After proving that triangle $EDP$ is isosceles, note that its exterior angles are equal: $\\angle APD=\\angle BED$. From this, triangles $ADP$ and $BED$ are congruent by the two indicated angles and sides $(AP=EB, PD=ED)$. Therefore, $BD=AD$, and triangle $ABD$ is equilateral. Thus, $\\angle BAD=60^{\\circ}$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12252,
        "difficulty": null
      },
      {
        "id": "7c54d1653b2d01317842d7584d439dc628be628ca996e816da283983ca42fd7e",
        "domain": "numina",
        "question": "$2+$ $[$ higher degree equations (miscellaneous). ]\n\nSolve the equation $\\left[x^{3}\\right]+\\left[x^{2}\\right]+[x]=\\{x\\}-1$.\n\n#",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 67744,
        "difficulty": null
      },
      {
        "id": "4020fb7d32dc0cf9002b0b8c40d74a56659cd5c725f6b6c18e84902236c964ca",
        "domain": "numina",
        "question": "3. A store bought a batch of greeting cards at 21 cents each, and sold them for a total of 14.57 yuan. If each card was sold at the same price, and the selling price did not exceed twice the purchase price, then the store made a profit of $\\qquad$yuan.",
        "answer": "4.7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 102488,
        "difficulty": null
      },
      {
        "id": "3bdc73a6a5e81364bbb03f9f6d73614638493b5eaa113e14f4d539c156afe5b5",
        "domain": "numina",
        "question": "A triangle $A B C$ has an area equal to 944. Let $D$ be the midpoint of $[A B]$, $E$ the midpoint of $[B C]$, and $F$ the midpoint of $[A E]$. What is the area of $D E F$?",
        "answer": "118",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 78916,
        "difficulty": null
      },
      {
        "id": "742994d3671d8eeb6374726e916bc8ab54059b6db4cdd5049c0bd3a334bf5685",
        "domain": "numina",
        "question": "3. From Nizhny Novgorod to Astrakhan, a steamboat takes 5 days, and on the return trip - 7 days. How long will it take for rafts to float from Nizhny Novgorod to Astrakhan?",
        "answer": "35",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35717,
        "difficulty": null
      },
      {
        "id": "62340048c31847e9f68faa7e82139bf56212305ed603b523fe19af4c9c2af374",
        "domain": "numina",
        "question": "Let real numbers $a_{1}, a_{2}, \\cdots, a_{n} \\in[0,2]$, and define $a_{n+1}=a_{1}$, find the maximum value of $\\frac{\\sum_{i=1}^{n} a_{i}^{2} a_{i+1}+8 n}{\\sum_{i=1}^{n} a_{i}^{2}}$.",
        "answer": "4",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 119586,
        "difficulty": null
      },
      {
        "id": "fdb667b8705424dd9dc70746bc99b108b47ca19bc46bfdb0a86aa24dd2faaa4a",
        "domain": "numina",
        "question": "Example 1-25 5 boys, 3 girls sit around a circular table. If there are no additional requirements, then $Q(8,8)=7!=5040$. If it is required that boy $B_{1}$ does not sit next to girl $G_{1}$, how many arrangements are there? If it is required that the 3 girls do not sit next to each other, how many arrangements are there?",
        "answer": "1440",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 145531,
        "difficulty": null
      },
      {
        "id": "a0ec42294a5e4616152869403e3bddfe8956fdc0fedad94bbb9a6f2a90e37166",
        "domain": "numina",
        "question": "An Olympic sequence is a sequence $s_{1}, s_{2}, \\ldots, s_{2023}$ where each of the 2023 terms is equal to 1 or -1. A slowly increasing sequence is a sequence of integers $t_{1}, t_{2}, \\ldots, t_{n}$ such that $1 \\leqslant t_{1}<t_{2}<\\cdots<t_{n} \\leqslant 2023$, and each of the $n-1$ differences $t_{i+1}-t_{i}$ is 1 or 2.\nFind the largest integer $C$ such that, for any Olympic sequence $s_{1}, s_{2}, \\ldots, s_{2023}$, there exists an integer $n \\geqslant 1$ and a slowly increasing sequence $t_{1}, t_{2}, \\ldots, t_{n}$ of length $n$ such that\n\n$$\n\\left|s_{t_{1}}+s_{t_{2}}+\\cdots+s_{t_{n}}\\right| \\geqslant C\n$$",
        "answer": "507",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 195790,
        "difficulty": null
      },
      {
        "id": "5088251bdc73c933650683b6109c371626ed5fac9c85c8d3220c559767430ba3",
        "domain": "numina",
        "question": "2. Let $\\triangle A B C$ be an obtuse isosceles triangle with base $\\overline{A B}$. On the base, there is a point $D$ such that $|\\angle A C D|=35^{\\circ}$. On the leg $\\overline{B C}$, there is a point $E$ such that $|C E|=|C D|$. Determine $|\\angle B D E|$.",
        "answer": "17.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87052,
        "difficulty": null
      },
      {
        "id": "4abd666dfcd0d21ce3cc8efa2a4ae98aaeadeb78ef4df5b575c809dcd8b0145b",
        "domain": "numina",
        "question": "4. Find all natural numbers $n$ for which all numbers $n+1, n+3, n+7$, $n+9, n+13$ and $n+15$ are prime.",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 161467,
        "difficulty": null
      },
      {
        "id": "29492571d95da0aefe34b33ffa7f10e17efdd848554cbd7008eb661bc7ddd3ed",
        "domain": "numina",
        "question": "4. We have\n\n$$\n\\begin{aligned}\n& a_{n}=\\prod_{k=2}^{n} \\frac{k^{2}}{k-1}=\\frac{2 \\cdot 2}{1} \\cdot \\frac{3 \\cdot 3}{2} \\cdot \\ldots \\ldots . \\cdot \\frac{n \\cdot n}{n-1}=\\frac{n!\\cdot n!}{(n-1)!}=n!\\cdot n=n!\\cdot[(n+1)-1]=(n+1)!-n! \\\\\n& b_{n}=\\sum_{k=2}^{n}[(k+1)!-k!]=(n+1)!-2, c_{n}=\\frac{(n+1)!-2}{(n+1)!}=1-\\frac{2}{(n+1)!} \\\\\n& d_{n}=\\left(1+\\frac{-2}{(n+1)!}\\right)^{(n+1)!}\n\\end{aligned}\n$$\n\n$$\n\\begin{gathered}\n\\text { We have } \\lim _{n \\rightarrow \\infty} c_{n}=\\lim _{n \\rightarrow \\infty}\\left(1+\\frac{-2}{(n+1)!}\\right)=1+0=1 \\text { and } \\lim _{n \\rightarrow \\infty} d_{n}=\\lim _{n \\rightarrow \\infty}\\left(1+\\frac{-2}{(n+1)!}\\right)^{(n+1)!}= \\\\\n=\\lim _{n \\rightarrow \\infty}\\left[\\left(1+\\frac{-2}{(n+1)!}\\right)^{\\frac{(n+1)!}{-2}}\\right]^{\\frac{-2}{(n+1)!}(n+1)!}=e^{-2}=\\frac{1}{e^{2}} \\quad \\text {.................................................................................... }\n\\end{gathered}\n$$\n\n## MATHEMATICS OLYMPIAD  Local stage, 16.02.2013  11th Grade\n\n## Problems:",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178451,
        "difficulty": null
      },
      {
        "id": "1ffef8fdba78a2ca8a8d1a7d9f76dd062260a14cf1e2f9e84fce9e78a1c248ba",
        "domain": "numina",
        "question": "3. (8 points) Longteng Primary School Grade 5 has four classes. Class 5-1 has 42 students, Class 5-2 is $\\frac{6}{7}$ of Class 5-1, Class 5-3 is $\\frac{5}{6}$ of Class 5-2, and Class 5-4 is 1.2 times the number of students in Class 5-3. The total number of students in Grade 5 is.",
        "answer": "144",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142103,
        "difficulty": null
      },
      {
        "id": "6718fee934868f4cf428493089da972c82697ca741e088e8b64189054b115d35",
        "domain": "numina",
        "question": "4. On each of the lines $y=0$ and $y=2$, there are 64 points with abscissas $1,2,3, \\ldots, 64$. In how many ways can three points be chosen from the marked 128 points so that they form the vertices of a right triangle?",
        "answer": "8420",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 3320,
        "difficulty": null
      },
      {
        "id": "3a0415da65fc3c6066d54a0703b67ca86eea5a03514ed18a219fdf14684878dd",
        "domain": "numina",
        "question": "## Task 3 - V00703\n\nThe tenth part of a number is increased by 3. The same value results when one hundredth of this number is decreased by 6!\n\nWhat is it?",
        "answer": "-100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183441,
        "difficulty": null
      },
      {
        "id": "1bfd7649ab295d030499ffe87e381271ed70f12db106d7949f3174182d1c97cd",
        "domain": "numina",
        "question": "Putnam 1992 Problem A2 Let the coefficient of x 1992 in the power series (1 + x) α = 1 + αx + ... be C(α). Find ∫ 0 1 C(-y-1) ∑ k=1 1992 1/(y+k) dy.",
        "answer": "1992",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 149400,
        "difficulty": null
      },
      {
        "id": "8fbca8ba0b5ce951f3ddd7012040966798b3f0d1813647315e9cf0535a092c47",
        "domain": "numina",
        "question": "4. Determine all three-digit numbers with distinct digits that are divisible by 6 and whose sum of digits is divisible by 6. How many such numbers are there?",
        "answer": "54",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86605,
        "difficulty": null
      },
      {
        "id": "891ae3b0aff48f80f9d65358f09797e3d57ad61123b6500ea92b870391318156",
        "domain": "numina",
        "question": "## Task 15/73\n\nHow many different line segments are necessary to draw the pattern shown in the figure without tracing any segment more than once?\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_1d2be4837fb839c2b31dg-234.jpg?height=306&width=437&top_left_y=972&top_left_x=1252)",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 182370,
        "difficulty": null
      },
      {
        "id": "8706a34172598c0f8345c94223340712924ff7fe270a57982933b5e4daf6b0b4",
        "domain": "numina",
        "question": "10.7. For natural numbers $a>b>1$, define the sequence $x_{1}, x_{2}, \\ldots$ by the formula $x_{n}=\\frac{a^{n}-1}{b^{n}-1}$. Find the smallest $d$ such that this sequence does not contain $d$ consecutive terms that are prime numbers, for any $a$ and $b$.\n\n(V. Senderov)",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17445,
        "difficulty": null
      },
      {
        "id": "8ba8c308bf73abeb0580447a28621f71dbb1d96aaa3ad3592ed94d1b1e1e7705",
        "domain": "numina",
        "question": "13. How many positive integers $x$, less than 10000, are there for which the difference $2^{x}-x^{2}$ is not divisible by 7?",
        "answer": "7142",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 23892,
        "difficulty": null
      },
      {
        "id": "0141b8e32aa018211198f715545c6728f6a96529937cedf8bfb10a1ecdaa0b79",
        "domain": "numina",
        "question": "Little Abel received as a gift a $2 \\times n$ board and $n$ tiles of size $2 \\times 1$. For example, the following figure shows the case where $n=10$, that is, when Abel has a $2 \\times 10$ board and 10 tiles of size $2 \\times 1$.\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-15.jpg?height=404&width=996&top_left_y=553&top_left_x=630)\n\nHe plays by filling the board using the $n$ tiles. For example, for $n=10$ Abel could fill it in the ways illustrated below:\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-15.jpg?height=580&width=1042&top_left_y=1144&top_left_x=608)\n\nHowever, observe that there are many other ways in which Abel can fill his board.\n\na) Calculate the total number of ways in which Abel can fill his board in the cases where $n=1, n=2$, and $n=3$, that is, in the cases where the boards have dimensions $2 \\times 1, 2 \\times 2$, and $2 \\times 3$.\n\nb) Let $a_{n}$ be the number of ways in which Abel can fill a $2 \\times n$ board using $n$ tiles of size $2 \\times 1$. Show that $a_{10}=a_{9}+a_{8}$.\n\nc) Calculate the total number of ways in which Abel can fill his board when $n=10$.",
        "answer": "89",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73876,
        "difficulty": null
      },
      {
        "id": "847f5d08d9e7af657575acbde52820c25a40a240180d048ae2256241d3489997",
        "domain": "numina",
        "question": "3. (7 points) There are 6 blue, 7 red, and 9 white bulbs. In how many ways can they be arranged into a garland (using all the bulbs) so that no two white bulbs are adjacent?",
        "answer": "3435432",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19428,
        "difficulty": null
      },
      {
        "id": "a67a2bf8ce3dc6b720b05f0f01b1b543c2c3a275916349c9682f23cfc9a156e4",
        "domain": "numina",
        "question": "11.4. Solve the equation:\n\n$$\n\\begin{gathered}\n\\frac{10}{x+10}+\\frac{10 \\cdot 9}{(x+10)(x+9)}+\\frac{10 \\cdot 9 \\cdot 8}{(x+10)(x+9)(x+8)}+\\cdots+ \\\\\n+\\frac{10 \\cdot 9 \\ldots 2 \\cdot 1}{(x+10)(x+9) \\ldots(x+1)}=11\n\\end{gathered}\n$$",
        "answer": "-\\frac{1}{11}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17975,
        "difficulty": null
      },
      {
        "id": "46d8c5821b512b85dc3c7b7919bb214b916892ccd7131490354e9fc84f13b81e",
        "domain": "numina",
        "question": "40. Find a five-digit number that has the following property: when multiplied by 9, the result is a number composed of the same digits but in reverse order, i.e., the number is reversed.",
        "answer": "10989",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 34771,
        "difficulty": null
      },
      {
        "id": "ce9b27f29d2f3276198a13c8b63c800e6ec91a94c66b22ef24470e61b329e00f",
        "domain": "numina",
        "question": "1. The number n is the product of three (not necessarily distinct) prime numbers. If we increase each of them by 1, the product of the increased numbers will be 963 more than the original product. Determine the original number $n$.",
        "answer": "2013",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75589,
        "difficulty": null
      },
      {
        "id": "986de247833fb6261adf18818942a5ee5cbb9482f5b474e00c49441aefaf5830",
        "domain": "numina",
        "question": "Exercise 7. An integer $k>1$ is said to be superb if there exist $m, n$, and $a$, three strictly positive integers such that\n\n$$\n5^{\\mathrm{m}}+63 \\mathrm{n}+49=\\mathrm{a}^{\\mathrm{k}}\n$$\n\nDetermine the smallest superb integer.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 77679,
        "difficulty": null
      },
      {
        "id": "8b5f0dc7da05aa5c685c9352b50a7f4cd951577178d7c96003d83123f809453c",
        "domain": "numina",
        "question": "24. If $x, y$ and $z$ are real numbers such that $x+y+z=9$ and $x y+y z+z x=24$, find the largest possible value of $z$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160498,
        "difficulty": null
      },
      {
        "id": "51e2906726b7d266d4232a41e1531cd88ac0569a26d1c83fddeb5fa2df93a3c0",
        "domain": "numina",
        "question": "13.402 The capacities of three vessels $A$, B, C, each of which has the shape of a cube, are in the ratio $1: 8: 27$, and the volumes of water poured into them are in the ratio 1:2:3. After transferring part of the water from vessel A to vessel B and from vessel B to vessel C, the water level in all three vessels became the same. Then, $128 \\frac{4}{7}$ liters of water were transferred from vessel C to vessel B, and after that, from vessel B to vessel A, so that the water depth in vessel A became twice that in vessel B. As a result, it turned out that there was now $10 \\theta$ liters less water in vessel A than there was initially. How much water was there initially in each vessel?",
        "answer": "500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 52247,
        "difficulty": null
      },
      {
        "id": "968874f03009724ca730185344cc95a8f6ecbd384a8e319069aeca2e788e1e5c",
        "domain": "numina",
        "question": "12. As shown in the figure, with the midpoint $O$ of the line segment $A B$ of length $4 \\mathrm{~cm}$ as the center and a radius of $2 \\mathrm{~cm}$, draw a circle intersecting the perpendicular bisector of $A B$ at points $E$ and $F$. Then, with $B$ and $A$ as centers and a radius of $4 \\mathrm{~cm}$, draw circular arcs intersecting the ray $A E$ at point $C$ and the ray $B E$ at point $D$. Finally, with $E$ as the center and $D E$ as the radius, draw the circular arc $D C$. Find the area of the \"oval\" $A F B C D A$ formed by these 4 real curve arcs. (Use $\\pi$ for the value of pi, do not use an approximate value)",
        "answer": "(12-4\\sqrt{2})\\pi-4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 96028,
        "difficulty": null
      },
      {
        "id": "fb9b0104e6a12d4637e97c9f85375d7edfe7e7e7cf517b9804225bb780e5d1ee",
        "domain": "numina",
        "question": "2.110. $\\frac{\\sqrt{c-d}}{c^{2} \\sqrt{2 c}} \\cdot\\left(\\sqrt{\\frac{c-d}{c+d}}+\\sqrt{\\frac{c^{2}+c d}{c^{2}-c d}}\\right) ; \\quad c=2 ; d=1 / 4$.",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50461,
        "difficulty": null
      },
      {
        "id": "645a4a73cf4fcd07af023eece5385143e97e0ca7c388183e11ec314f550d5c6e",
        "domain": "numina",
        "question": "Problem 3. In the school, there are 50 teachers, of whom 29 drink coffee, 28 drink tea, and 16 do not drink either coffee or tea. How many teachers drink only coffee, and how many drink only tea?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 188941,
        "difficulty": null
      },
      {
        "id": "cb1fcf1c6f8a08eea85ab1f158b80931cdb3e28a687778ffd5ff97a1921e4fb7",
        "domain": "numina",
        "question": "A plane can intersect at most how many edges of a regular 10-sided prism?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170048,
        "difficulty": null
      },
      {
        "id": "7e00e59cf1574d21bef7a4086acd4d23de3be262bac1c40b632b10f01356633d",
        "domain": "numina",
        "question": "At the edge of a circular lake, there are stones numbered from 1 to 10, in a clockwise direction. Frog starts from stone 1 and jumps only on these 10 stones in a clockwise direction.\n\na) If Frog jumps 2 stones at a time, that is, from stone 1 to stone 3, from stone 3 to stone 5, and so on, on which stone will Frog be after 100 jumps?\n\nb) If on the first jump, Frog goes to stone 2, on the second jump to stone 4, on the third jump to stone 7, that is, in each jump he jumps one more stone than in the previous jump. On which stone will Frog be after 100 jumps?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 73788,
        "difficulty": null
      },
      {
        "id": "f7480e545721380f2b1329acbc5d1d716e6b3a31a1415bd3f72032d316973b59",
        "domain": "numina",
        "question": "Example 3 (36th Canadian Mathematical Olympiad) Let $T$ be the set of all positive divisors of $20044^{100}$, and let $S$ be a subset of $T$ such that no element of $S$ is an integer multiple of any other element of $S$. Find the maximum possible number of elements in $S$.\n\nThe set $T$ consists of all positive divisors of $20044^{100}$. A subset $S$ of $T$ is such that no element of $S$ is an integer multiple of any other element of $S$. Determine the maximum possible number of elements in $S$.",
        "answer": "10201",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136365,
        "difficulty": null
      },
      {
        "id": "0fe056d30f830accad2e5094809a289fb657a1b25a8a10519d98eb17628bd953",
        "domain": "numina",
        "question": "Example 8 (40th IMO National Team Selection Test) For non-negative real numbers $x_{1}, x_{2}$, $\\cdots, x_{n}$ satisfying the condition $\\sum_{i=1}^{n} x_{i}=1$, find the maximum value of $\\sum_{i=1}^{n}\\left(x_{j}^{4}-x_{j}^{5}\\right)$.",
        "answer": "\\frac{1}{12}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 132103,
        "difficulty": null
      },
      {
        "id": "4c0dd8f0806acaeab20350d3193decab75c6998026c5cc7614e865bc603a5993",
        "domain": "numina",
        "question": "Yatsenko I.V.\n\nVanya thought of a simple three-digit number, all digits of which are different.\n\nWhat digit can it end with if its last digit is equal to the sum of the first two?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 70142,
        "difficulty": null
      },
      {
        "id": "e2aea8e00bd51a4d8ad367ad7019f091176ece8b758465328556af5f1a2b1948",
        "domain": "numina",
        "question": "11.4. In a semicircle with a radius of 18 cm, a semicircle with a radius of 9 cm is constructed on one of the halves of the diameter, and a circle is inscribed, touching the larger semicircle from the inside, the smaller semicircle from the outside, and the second half of the diameter. Find the radius of this circle.\n\n#",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9490,
        "difficulty": null
      },
      {
        "id": "2f900c3c12504fde209b0fd6719dab0d6bba13214e01e60672950cfc6cd8471a",
        "domain": "numina",
        "question": "9.5. What is the maximum number of figures consisting of 4 1x1 squares, as shown in the diagram, that can be cut out from a $6 \\times 6$ table, if cutting can only be done along the grid lines?\n\n#",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 15854,
        "difficulty": null
      },
      {
        "id": "1c43cbaf8911ce2c6634689b683064aebdfed4ec4659e5e94c040113e807441f",
        "domain": "numina",
        "question": "We randomly select four points independently on the surface of a sphere. What is the probability that the tetrahedron defined by them contains the center of the sphere?",
        "answer": "\\frac{1}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170078,
        "difficulty": null
      },
      {
        "id": "a8d05dec8dd36893badf6c74f2338313bf8241dc60d168c70f5fc23a67c55834",
        "domain": "numina",
        "question": "Seven dwarfs stood around the perimeter of their garden, one in each corner, and stretched a rope around the entire garden. Snow White came out from Smudla and walked along the rope. First, she walked four meters to the east, where she met Prof. From there, she continued two meters to the north until she reached Rejpal. From Rejpal, she walked west and after two meters, she met Stydlín. She then continued three meters to the north, until she reached Štístko. She set off to the west and after four meters, she met Kejchal, from where she had three meters to the south to Dřímal. Finally, she followed the rope to the shortest path back to Smudla, thus completing her walk around the garden.\n\nHow many square meters does the entire garden have?\n\n(M. Mach)\n\nHint. Draw the shape of the garden, preferably on graph paper.",
        "answer": "22",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76488,
        "difficulty": null
      },
      {
        "id": "fcbae05d57d4563b8f146077d1b3e7b4e75a89c5e6ef4d5056e7dd9fcbf09705",
        "domain": "numina",
        "question": "1. What is the minimum number of cells that need to be marked in a $20 \\times 20$ table so that each vertical or horizontal strip of $1 \\times 12$ contains at least one marked cell.",
        "answer": "32",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 8310,
        "difficulty": null
      },
      {
        "id": "03dd86adbec5012e25471d6fcf7ef1efb1019366751cef9858b58757504a7f4c",
        "domain": "numina",
        "question": "5. Point $M$ divides the diagonal $A C$ of square $A B C D$ in the ratio $A M: M C=2: 1$. A line passing through point $M$ divides the square into two parts, the areas of which are in the ratio $9: 31$. In what ratio does this line divide the perimeter of the square?",
        "answer": "\\frac{27}{53}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 2405,
        "difficulty": null
      },
      {
        "id": "9d884b01a2be76365601bc760f51fbf0bec41e65f3fb77ceddc0587dec804c37",
        "domain": "numina",
        "question": "5. (10 points) The product of two decimal numbers, when rounded to the nearest tenth, is 27.6. It is known that both decimals have one decimal place and their units digits are both 5. What is the exact product of these two decimals? $\\qquad$ .",
        "answer": "27.55",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93800,
        "difficulty": null
      },
      {
        "id": "690b4671fbfc8c0a36e0aab416655f4c7291582a42f1fd7a7f5e29a67ebc57df",
        "domain": "numina",
        "question": "7.2. There are 11 kg of cereal. How can you measure out 1 kg of cereal using two weighings on a balance scale, if you have one 3 kg weight?",
        "answer": "1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22542,
        "difficulty": null
      },
      {
        "id": "ae6c85e28ed73176f422ba024db37a1df46e2c02e483b1fc84b5a5e7ae2542be",
        "domain": "numina",
        "question": "The graph of the function $y=|x-1|+|x+1|$ and the line $y=c$ enclose a region with an area of 30. What is the value of the constant $c$?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170949,
        "difficulty": null
      },
      {
        "id": "d75afc0ce9e437c9120e25a59fd982608ed7eba844791886f080a73286ad5995",
        "domain": "numina",
        "question": "In the drawing to the side, triangle $ABC$ is equilateral and $BD = CE = AF = \\frac{AB}{3}$. The ratio $\\frac{EG}{GD}$ can be written in the form $\\frac{m}{n}, \\operatorname{gcd}(m, n)=1$. What is the value of $m+n$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_12f29789e10e6749ebb9g-37.jpg?height=460&width=516&top_left_y=1150&top_left_x=884)\n\n#",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73477,
        "difficulty": null
      },
      {
        "id": "614b119c949e9d84da04999a9b1c9de0a1bf83510300336718eb738766b51d42",
        "domain": "numina",
        "question": "12. Find the smallest natural number greater than 1 that is at least 600 times greater than each of its prime divisors.",
        "answer": "1944",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 37295,
        "difficulty": null
      },
      {
        "id": "9b9e8afa61a7c1d474ad458a217b577b99c2ca66754b77e65d35b48edab114e1",
        "domain": "numina",
        "question": "10. If real numbers $a, b, c, d, e$ satisfy $a+b+c+d+e=8, a^{2}+b^{2}+c^{2}+d^{2}+e^{2}=16$, then the sum of the maximum and minimum values of $e$ is $\\qquad$",
        "answer": "\\frac{16}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120443,
        "difficulty": null
      },
      {
        "id": "cb29f9fb0769f536a1aaca1fb09e3e33a8b40412767614e3ed26204bee4f2e8b",
        "domain": "numina",
        "question": "10th Australian 1989 Problem B3 Let N be the positive integers. The function f : N → N satisfies f(1) = 5, f( f(n) ) = 4n + 9 and f(2 n ) = 2 n+1 + 3 for all n. Find f(1789). Solution",
        "answer": "3581",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 147782,
        "difficulty": null
      },
      {
        "id": "9b514ee2fc1faa07fa2d82a4e4e9bc4d7b0ad631e236d53b362b1eadffaca5ee",
        "domain": "numina",
        "question": "Let $A B C$ be an isosceles triangle with $A C=B C$. The point $D$ lies on the side $A B$ such that the semicircle with diameter $[B D]$ and center $O$ is tangent to the side $A C$ in the point $P$ and intersects the side $B C$ at the point $Q$. The radius $O P$ intersects the chord $D Q$ at the point $E$ such that $5 \\cdot P E=3 \\cdot D E$. Find the ratio $\\frac{A B}{B C}$.",
        "answer": "\\frac{6}{5}",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 197616,
        "difficulty": null
      },
      {
        "id": "154e7ecfef6e1fc566219f8be32dea7e9150eba909fc58be2b8e302f56d1da80",
        "domain": "numina",
        "question": "4. The four-digit number $\\overline{a b c d}$ is divisible by 3, and $a, b, c$ are permutations of three consecutive integers. Then the number of such four-digit numbers is\n$\\qquad$ .",
        "answer": "184",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 116841,
        "difficulty": null
      },
      {
        "id": "d00b67ddb7a94bbb4d0540b02c517fb45d205cd7e011396962739fa683b0f468",
        "domain": "numina",
        "question": "13. Sarah and Hagar play a game of darts. Let $O_{0}$ be a circle of radius 1. On the $n$th turn, the player whose turn it is throws a dart and hits a point $p_{n}$ randomly selected from the points of $O_{n-1}$. The player then draws the largest circle that is centered at $p_{n}$ and contained in $O_{n-1}$, and calls this circle $O_{n}$. The player then colors every point that is inside $O_{n-1}$ but not inside $O_{n}$ her color. Sarah goes first, and the two players alternate turns. Play continues indefinitely. If Sarah's color is red, and Hagar's color is blue, what is the expected value of the area of the set of points colored red?",
        "answer": "\\frac{6\\pi}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151257,
        "difficulty": null
      },
      {
        "id": "aca836cb9cefa3bfc396d3f52084ac59d6f8696f615a6035995e719555f37de4",
        "domain": "numina",
        "question": "5. In the number $2016 * * * * 02 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,2,4,6,7,8$ (digits can be repeated) so that the resulting 11-digit number is divisible by 6. In how many ways can this be done?",
        "answer": "2160",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 3625,
        "difficulty": null
      },
      {
        "id": "1403835cebeed46b837bb3968053cce8f2a4706230742dbeb9c1f167073a5382",
        "domain": "numina",
        "question": "1. Solve the equation: $(x-2016)(x-2017)(x-2018)=(x-2017)(x-2018)(x-2019)$.",
        "answer": "20172018",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13186,
        "difficulty": null
      },
      {
        "id": "39ad84268d8b1b1ff758a26ce6184c8694c555fdd837fcbed4b01bd480c156c3",
        "domain": "numina",
        "question": "1B. The binomial coefficients of the second, third, and fourth terms in the expansion of the binomial $(1+x)^{n}$ form an arithmetic progression. Calculate $n$.",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 189989,
        "difficulty": null
      },
      {
        "id": "23f0723390905eba07619ec4540cdbae662821ed5b9e6080733bfa39b89bacb2",
        "domain": "numina",
        "question": "96. Let $X$ be the number of points that fall when a die is rolled once. Find the variance of the random variable $X$.",
        "answer": "\\frac{35}{12}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32282,
        "difficulty": null
      },
      {
        "id": "0e6ab0b26e0f36105740514c988f75f649109bf00f403459915fc9927aa5c114",
        "domain": "numina",
        "question": "$3 \\cdot 3$ Simplify the fraction $\\frac{x^{8}+x^{6}+x^{4}+x^{2}+1}{x^{4}+x^{3}+x^{2}+x+1}$, and calculate the value of the fraction when $x=-0.02$.",
        "answer": "\\frac{312499999}{306250000}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146066,
        "difficulty": null
      },
      {
        "id": "69c9e98c99a6edeb7cbec7f93ccbcf48ad999b6be10110f4018174190d908f70",
        "domain": "numina",
        "question": "3. In triangle ABC, the median BM is drawn. It is known that $\\angle A B M=40^{\\circ}$, and $\\angle C B M=70^{\\circ}$. Find the ratio $A B: B M$.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7032,
        "difficulty": null
      },
      {
        "id": "268e73ae8f99153918566ce3540e8814641b46a44c93975c29337ef6ac8c51b0",
        "domain": "numina",
        "question": "10. There are 4 identical squares. By joining them edge-to-edge without overlapping, they can form $\\qquad$ different plane figures. (Figures that can coincide after translation, rotation, or reflection are considered the same figure)",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 102392,
        "difficulty": null
      },
      {
        "id": "248e54ff0fc6d0ee3132a3d25ad7adeb6c600e20fd3311327b9ed0b8803356ad",
        "domain": "numina",
        "question": "G1.2 Let $a_{1}, a_{2}, a_{3}, \\ldots$ be an arithmetic sequence with common difference 1 and $a_{1}+a_{2}+a_{3}+\\ldots+a_{100}=2012$. If $P=a_{2}+a_{4}+a_{6}+\\ldots+a_{100}$, find the value of $P$.",
        "answer": "1031",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153299,
        "difficulty": null
      },
      {
        "id": "89f6361684ff4766bad88b822c8d56bbd2eff0b01c1f4b8fa0730f0a3db7abfc",
        "domain": "numina",
        "question": "Russian 1999 Problem 1 The digits of n strictly increase from left to right. Find the sum of the digits of 9n.",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 148243,
        "difficulty": null
      },
      {
        "id": "06a94c4f34cab8256846c4c73dd981d072eb69a4a850458e0c20f5f02a9e5455",
        "domain": "numina",
        "question": "564. $y=x^{2}+2$.\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly.\n\n564. $y=x^{2}+2$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32064,
        "difficulty": null
      },
      {
        "id": "78c4bd9ce3031f2920bcdb4cd9238bb8a537d7604db5bebb706c7f2da6be838c",
        "domain": "numina",
        "question": "5. The complex numbers $z_{1}, z_{2}$ satisfy\n$$\n\\begin{array}{l}\n\\left|\\left(\\left|z_{1}\\right| z_{1}\\right)+\\left(\\left|z_{2}\\right| z_{2}\\right)\\right| \\\\\n=\\left|\\left(\\left|z_{1}\\right| z_{2}\\right)+\\left(z_{1}\\left|z_{2}\\right|\\right)\\right|,\n\\end{array}\n$$\n\nand $\\left|z_{1}-\\frac{1}{z_{2}}\\right|+\\left|z_{2}-\\frac{1}{z_{1}}\\right|$ has a maximum value and a minimum value whose sum is $4 \\sqrt{3}$. Then $\\left(\\left|z_{1} \\overline{z_{2}}\\right|+\\left|z_{2} \\overline{z_{1}}\\right|\\right)^{2016}=$",
        "answer": "6^{2016}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117394,
        "difficulty": null
      },
      {
        "id": "b78eda3cb1bafffb163d66324b3abe93a8acb817738917df28b43b75d35a2808",
        "domain": "numina",
        "question": "1. Ivan Ivanovich approached a source with two empty cans; one held 10 liters, and the other held 8 liters. Water from the source flowed in two streams - one stronger, the other weaker. Ivan Ivanovich simultaneously placed the cans under the streams and, when half of the smaller can was filled, he switched the cans. To Ivan Ivanovich's surprise, the cans filled up simultaneously. How many times more water does the stronger stream provide compared to the weaker one?",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 21785,
        "difficulty": null
      },
      {
        "id": "fde92c48c88db063ea2f28c5890cbbb4f065c316129331245be32760eab3991d",
        "domain": "numina",
        "question": "3. Given a triangle $ABC$ with the largest side $BC$. The bisector of its angle $C$ intersects the altitudes $AA_{1}$ and $BB_{1}$ at points $P$ and $Q$ respectively, and the circumcircle of $ABC$ at point $L$. Find $\\angle ACB$, if it is known that $AP=LQ$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7973,
        "difficulty": null
      },
      {
        "id": "efc658bb974d1507a56da18b3d22b263dd308aa8c7fc0d0fab9bbf8e90079671",
        "domain": "numina",
        "question": "## Zadatak B-4.1.\n\nRiješite jednadžbu\n\n$$\n\\log _{2020} \\sqrt{x}+\\log _{2020} \\sqrt[4]{x}+\\log _{2020} \\sqrt[8]{x}+\\ldots+\\log _{2020} \\sqrt[2^{n}]{x}+\\ldots=\\frac{2}{\\log _{2} x}+\\log _{x} 505\n$$",
        "answer": "2020",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87875,
        "difficulty": null
      },
      {
        "id": "3d56dda1efb4fc7a36ad9acef26b9f21bda0a6e215d233d15ab9d7f36fce4951",
        "domain": "numina",
        "question": "66. In the winter basketball games, player Xiao Ming scored 23 points, 14 points, 11 points, and 20 points in the sixth, seventh, eighth, and ninth games, respectively. His average score in the first nine games is higher than his average score in the first five games. If his average score in the first ten games is higher than 18 points, then his score in the tenth game must be at least $\\qquad$ points.",
        "answer": "29",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101300,
        "difficulty": null
      },
      {
        "id": "a749ea7aae27f90e353f5ba315ff45a9ae8f86b855ed49cb46e4bb5e352686e7",
        "domain": "numina",
        "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 6, 24, and 24. A truncated cone is placed on the table with its smaller base down, and it shares a generatrix with each of the other cones. Find the radius of the smaller base of the truncated cone.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7670,
        "difficulty": null
      },
      {
        "id": "669af18db204d1af582a49be08a2c0ef2ed6cae86132fa0c54a2415d28a7fe5a",
        "domain": "numina",
        "question": "10. The common difference of an arithmetic sequence with real terms is 4, and the square of the first term plus the sum of the remaining terms does not exceed 100. Such a sequence can have at most $\\qquad$ terms.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99554,
        "difficulty": null
      },
      {
        "id": "5345be7a7f6e3e7ef303dab75f11a8a533b808883eae228c9bef1471d5039a9f",
        "domain": "numina",
        "question": "10.419 A circle with its center on side $AB$ of triangle $ABC$ touches the other two sides of the triangle. Find the area of the circle if $a=13$ cm, $b=14$ cm, $c=15 \\mathbf{cm}$, where $a, b$ and $c$ are the lengths of the sides of the triangle.",
        "answer": "\\frac{3136}{81}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 52105,
        "difficulty": null
      },
      {
        "id": "877314c4fea7301a28bf2a0c70155f268aa87e0fbb5fa37a6e21f4ebecdaf414",
        "domain": "numina",
        "question": "4. Let $f(t)=\\sqrt{1+t^{2}}-t$. Calculate the value of the expression\n\n$$\nf(x) f(y)+f(y) f(z)+f(z) f(x)\n$$\n\nif $x>0, y>0, z>0$ and $x y+y z+z x=1$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 187718,
        "difficulty": null
      },
      {
        "id": "74d1a3103358e58d13f91508afb326e1198704f687e55e5d94f85791ee9026df",
        "domain": "numina",
        "question": "[ [ $\\left[\\begin{array}{l}\\text { Divisibility of numbers. General properties } \\\\ \\text { p }\\end{array}\\right]$\n\nFind such an $n$ that the number $10^{n}-1$ is divisible by a) 7 ; b) 13 ; c) 91 ; d) 819.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69615,
        "difficulty": null
      },
      {
        "id": "0cebfa2eb405f50e6682ab69b6dbd635b4b5a72476c22057f3bfed5cfd1f4e3d",
        "domain": "numina",
        "question": "## Task 1 - 090621\n\nKlaus participated in a track race as a member of the cycling section of a company sports club. After the race, his brother Reiner asked Klaus about the outcome of the race. Klaus said:\n\n\"When I crossed the finish line, no rider was next to me; exactly one third of the participating riders had already reached the finish line, and exactly half of all participants were still behind me.\"\n\nState which position Klaus finished in this race and how many riders in total participated in the race!",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 183238,
        "difficulty": null
      },
      {
        "id": "d935f219e50380030b4c03b4b114fdf82be5e7e54a3b21d4150de1502bb9b0b8",
        "domain": "numina",
        "question": "Problem 4.5. Hooligan Dima made a construction in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the figure.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire construction to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
        "answer": "65",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17347,
        "difficulty": null
      },
      {
        "id": "7335e31fdc5442c4f9f829208dc7c6e86e9287a868876a0dc71e94ed0e56b72d",
        "domain": "numina",
        "question": "7. The equation $x^{2}-a|x|+a^{2}-3=0$ has a unique real solution for $x$, then $a=$ $\\qquad$ .",
        "answer": "-\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98557,
        "difficulty": null
      },
      {
        "id": "89f3eef8f059fef3a2e0de9b4167c079eccc13f307f9c3f3c917c44052697b57",
        "domain": "numina",
        "question": "9. [6] Five guys each have a positive integer (the integers are not necessarily distinct). The greatest common divisor of any two guys' numbers is always more than 1 , but the greatest common divisor of all the numbers is 1 . What is the minimum possible value of the product of the numbers?",
        "answer": "32400",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151163,
        "difficulty": null
      },
      {
        "id": "a53ddec9896e6461834d7cb341e29e9cfaa1a1f26e0ca75f99087fad6f704eae",
        "domain": "numina",
        "question": "15 If $m, n \\in\\left\\{x \\mid x=a_{2} \\times 10^{2}+a_{1} \\times 10+a_{0}\\right\\}$, where $a_{i} \\in\\{1,2,3, 4,5,6,7\\}, i=0,1,2$, and $m+n=636$, then the number of distinct points $(m, n)$ representing points on the plane is $\\qquad$",
        "answer": "90",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 111536,
        "difficulty": null
      },
      {
        "id": "6d953529e0191cb7f5eb6b1d63ead07fa22bf836ea4ae15861d82e26767f6307",
        "domain": "numina",
        "question": "Find the root of the following equation, with a precision of 3 significant digits.\n\n$$\n(\\sqrt{5}-\\sqrt{2})(1+x)=(\\sqrt{6}-\\sqrt{3})(1-x)\n$$",
        "answer": "-0.0678",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 164748,
        "difficulty": null
      },
      {
        "id": "36344a291f893e4dcda7bac29369fa16bea718f9817438448d0702fddbff5419",
        "domain": "numina",
        "question": "731. Calculate the approximate value:\n1) $1.08^{3.96}$;\n2) $\\frac{\\sin 1.49 \\cdot \\operatorname{arctg} 0.07}{2^{2.95}}$.",
        "answer": "0.01",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 34302,
        "difficulty": null
      },
      {
        "id": "44d62217679576955e5f63ae19224bee32666cedf125490eaf54c6b33854c309",
        "domain": "numina",
        "question": "It is known that the point symmetric to the center of the inscribed circle of triangle $ABC$ with respect to side $BC$ lies on the circumcircle of this triangle. Find angle $A$.\n\n#",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63493,
        "difficulty": null
      },
      {
        "id": "4274424a1df38f795aaddb7aaef05fe38f1da8bc1cf974ea903e632c9b635230",
        "domain": "numina",
        "question": "145. Three friends had a monkey. They bought a certain number of mango fruits and hid them. At night, one of the friends wanted to indulge, went to the storeroom, and wanted to take a third of the fruits, but it turned out that one fruit was left over, and he gave it to the monkey, and ate his share. After some time, the second one came and, not suspecting that the first one had already taken his share, divided the remainder into three equal parts, and again one fruit was left over, and he gave it to the monkey and ate his share. Finally, the third one came and also divided the remainder into three equal parts, and again one fruit was left over, which he gave to the monkey, and ate his share.\n\nThe next morning, all three came to the storeroom and divided the remaining fruits equally among themselves, and again one fruit was left over, which they gave to the monkey.\n\nDetermine the smallest number of fruits that satisfies the above conditions of distribution.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_926f0bb262c8f9569516g-029.jpg?height=268&width=394&top_left_y=1491&top_left_x=837)\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_926f0bb262c8f9569516g-030.jpg?height=297&width=1307&top_left_y=248&top_left_x=383)\n\n## Problem from the tractate \"Bava Batra\".\n\nTalmud.",
        "answer": "79",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 37787,
        "difficulty": null
      },
      {
        "id": "fcabf0418758858800a298d9a98c44fe6d10b4a05108cd2be15cc8456bcd77da",
        "domain": "numina",
        "question": "390. A circle is divided into five equal arcs by five points. Find the measure of each of them.",
        "answer": "72",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 43039,
        "difficulty": null
      },
      {
        "id": "10591a74da55037ad0fdee7aa2800686bbf2fbdf0475a3ecf99cdd37fd6f512d",
        "domain": "numina",
        "question": "Given a sequence $\\left\\{a_{n}\\right\\}$ with all terms no less than 1, it satisfies:\n$$\na_{1}=1, \\quad a_{2}=1+\\frac{\\sqrt{2}}{2}, \\quad\\left(\\frac{a_{n}}{a_{n+1}-1}\\right)^{2}+\\left(\\frac{a_{n}-1}{a_{n-1}}\\right)^{2}=2 .\n$$\n\nTry to find:\n(1) The general term formula of the sequence $\\left\\{a_{n}\\right\\}$;\n(2) The value of $\\lim _{n \\rightarrow+\\infty} \\frac{a_{n}}{n}$.",
        "answer": "\\frac{2}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112920,
        "difficulty": null
      },
      {
        "id": "5b6e4abd3f066443e7b1811fe83744aed981492e8e3dbbe0f7b4ab28aa958541",
        "domain": "numina",
        "question": "25. In triangle $ABC$, the altitude $BH$ is drawn, point $O$ is the center of the circumscribed circle around it, and the length of its radius is $R$. Find the measure in radians of the largest of the angles $ACB$ and $BAC$, given that $R=(5 / 8) \\cdot B H=(5 / 6) \\cdot O H$. If necessary, round the found value to two decimal places.",
        "answer": "1.82",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5647,
        "difficulty": null
      },
      {
        "id": "ccfa9f8acb2fcba29c4e22cf4fc11ac34ceced1d24935377723436e0e7be198a",
        "domain": "numina",
        "question": "99. How many diagonals does a convex decagon have?",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44679,
        "difficulty": null
      },
      {
        "id": "c57cefd75898abbcd00ae1dc9c36d75fbe3e111af0fa19e557cb9bd1d92c7af3",
        "domain": "numina",
        "question": "And $c$ is a constant), the difference in eccentricity between $C_{1}$ and $C_{2}$ does not exceed 1, and $C_{2}$ has an asymptote with a slope no less than $\\frac{4}{3}$,\n$C_{1}$ and $C_{2}$ intersect the positive $x$-axis at points $A$ and $B$, respectively, and the intersection point of the two curves in the first quadrant is $D$. Try to find the maximum area of $\\triangle A B D$ and the equations of $C_{1}$ and $C_{2}$ at this time.",
        "answer": "\\frac{9\\sqrt{5}}{50}^{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93442,
        "difficulty": null
      },
      {
        "id": "284b53c4fc7dc92871b91bc7d71d5548545428ceb79f3c043dd6456a6a060dbf",
        "domain": "numina",
        "question": "18. If in a convex decagon no three diagonals intersect at the same point inside the decagon, how many segments can the diagonals be divided into by their intersection points inside the decagon?",
        "answer": "455",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 112811,
        "difficulty": null
      },
      {
        "id": "ff9aa4b52f8839f620709dab8091ff07d50004cddbb77d0f31f1f5aa56918d5e",
        "domain": "numina",
        "question": "## Task 1 - 320731\n\na) Four red balls, two yellow balls, and one blue ball are to be distributed into two boxes $A$ and $B$ such that box $A$ contains three balls and box $B$ contains four balls.\n\nHow many such distributions are there in total?\n\nb) Now, balls of the same color are distinguished by an additional numbering. The distributions then differ not only in how many balls of each color are in boxes $A$ and $B$, but also in which numbers they carry.\n\nHow many such distributions are there in total?",
        "answer": "35",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183855,
        "difficulty": null
      },
      {
        "id": "631608b19521bf29f5277e8573b21e9234f16c9c7058d32688f6201569ca144a",
        "domain": "numina",
        "question": "[ Arithmetic. Mental calculation, etc.]\n\nFor the book, 100 rubles were paid, and it remains to pay as much as it would remain to pay if it had been paid as much as it remains to pay. How much does the book cost?",
        "answer": "200",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 69074,
        "difficulty": null
      },
      {
        "id": "ee10b4cbf3ed53dc14fcb0b56939380f42ae2217af76f548768b6b1e2543e6da",
        "domain": "numina",
        "question": "Ten points are marked around a circle, as illustrated in the figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_0308bd91a070e7a80796g-2.jpg?height=297&width=280&top_left_y=454&top_left_x=1068)\n\n(a) How many chords can be formed by connecting any two of these points? (A chord is a line segment connecting two points on a circle.)\n\n(b) How many triangles can be formed by connecting any three of these points?",
        "answer": "45",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74268,
        "difficulty": null
      },
      {
        "id": "0710cece084d35e684b95e289ef0e25720a78047040d28369d4104130d01cf27",
        "domain": "numina",
        "question": "55 (978). There are 4 balls of different masses. Using a balance scale without weights, how many weighings are needed to arrange these balls in order of decreasing mass?",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 35994,
        "difficulty": null
      },
      {
        "id": "2698ec97349b9fa3db1c27f9645aef9599d4a91af97230e5fe04866de72e7de1",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nIn the diagram, $\\triangle A B C$ is isosceles with $A B=B C$. If $\\angle B A C=t^{\\circ}$, what is the measure of $\\angle A B C$, in degrees?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_23a04f66c1167e793e90g-1.jpg?height=263&width=374&top_left_y=2053&top_left_x=903)",
        "answer": "76",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194432,
        "difficulty": null
      },
      {
        "id": "63a62be2aae313019c79d5e8667513c3bfd4c45a3fc159c4c94be4178b4cd419",
        "domain": "numina",
        "question": "Problem 8.3. In triangle $ABC$, the sides $AC=14$ and $AB=6$ are known. A circle with center $O$, constructed on side $AC$ as the diameter, intersects side $BC$ at point $K$. It turns out that $\\angle BAK = \\angle ACB$. Find the area of triangle $BOC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-24.jpg?height=378&width=422&top_left_y=1434&top_left_x=512)",
        "answer": "21",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 18115,
        "difficulty": null
      },
      {
        "id": "8770da86fc48f4cd2b8d4a1aed817b7606f32383a8b6343c803b3a8fef914e7e",
        "domain": "numina",
        "question": "(1) The area of the inscribed square in the ellipse $\\frac{x^{2}}{5^{2}}+\\frac{y^{2}}{3^{2}}=1$ is",
        "answer": "\\frac{450}{17}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105621,
        "difficulty": null
      },
      {
        "id": "c214f3ed0cfcffe3deb9bf8945fde45ed04dcacea6b22dd36a84e5d119534570",
        "domain": "numina",
        "question": "12. In the Tumba-Yumba tribe with a population of 30 people, a trader arrives. After studying the customs of the tribe, the trader proposes to play a game. For each natural exchange of goods conducted in the market by two tribespeople, the trader gives each participant one gold coin. If at the end of the day, two different tribespeople end up with the same number of coins, all goods and coins go to the trader. A tribesperson trades goods only with another known tribesperson, and the cunning trader knows this. By the end of the day, the trader finds a group of tribespeople with the same number of coins. The tribespeople are shocked, the trader is happy, and the offending tribespeople are expelled from the market. Then the Chief appears and offers the trader a rematch. Let the trader distribute 270 coins among the remaining tribespeople so that no one has the same number of coins (i.e., each tribesperson should have a unique number of coins). If the trader cannot do this, all the coins and goods remain in the tribe.\n\n- Is it true that in the original game, the tribespeople had no chance?\n- Is it true that the Chief can return all the goods and gold?\n- Is it true that the Chief knew how many tribespeople were expelled from the market?\n- What is the maximum number of tribespeople that needed to be expelled for the Chief to win?\n- Is it true that if the trader had lost the first game, the tribespeople could have lost the second game?\n\nExplain your answers.",
        "answer": "24",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 7472,
        "difficulty": null
      },
      {
        "id": "83b41946545cfc5d503d9eb39e037d098bc72efd9929bfae7ae0eb98b13d5248",
        "domain": "numina",
        "question": "4. From the 12 edges and the 12 face diagonals of a cube, select $n$ lines such that any two of the selected line segments are skew lines. The maximum value of $n$ is $\\qquad$ .",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 127639,
        "difficulty": null
      },
      {
        "id": "237e7598ae6293932298bbdf2dacde2b87797faff40d9ad2cf58c4fb31b1c965",
        "domain": "numina",
        "question": "Each of the thirty sixth-graders has one pen, one pencil, and one ruler. After their participation in the Olympiad, it turned out that 26 students lost a pen, 23 - a ruler, and 21 - a pencil. Find the smallest possible number of sixth-graders who lost all three items.",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57048,
        "difficulty": null
      },
      {
        "id": "7ce65f296618858aade7419ba09937e23138526093f52a3c2cacb9c83fa398c0",
        "domain": "numina",
        "question": "3.2 Two workers together produced 72 parts per shift. After the first worker increased their labor productivity by $15 \\%$, and the second by $25 \\%$, together they started producing 86 parts per shift. How many parts does each worker produce per shift after the increase in labor productivity?",
        "answer": "46",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49542,
        "difficulty": null
      },
      {
        "id": "08e046726a0e4460708903c7561f35b704171d97dbea7529a178880e19de81f3",
        "domain": "numina",
        "question": "4. The function $f(x, y)$ satisfies for all non-negative integers $x, y$:\n(1) $f(0, y)=y+1$;\n(2) $f(x+1,0)=f(x, 1)$;\n(3) $f(x+1, y+1)=f[x, f(x+1, y)]$.\nDetermine $f(4,1981)$.\n(IMO -22 Problem)",
        "answer": "2^{(1984)}-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125037,
        "difficulty": null
      },
      {
        "id": "6d43d47e03deadb0809d610b7a34d4fb359e012d92aafab396c601f6f691b918",
        "domain": "numina",
        "question": "【Question 3】As shown in the figure, points $C$, $E$, and $B$ are collinear, $C B \\perp A B$, $A E / / D C$, $A B=8$, $C E=5$, then the area of $\\triangle A E D$ is $\\qquad$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97643,
        "difficulty": null
      },
      {
        "id": "3bb9f1d9855cb9d775e202a452ec82042145e4a8e74c346f31fa8d6325f17582",
        "domain": "numina",
        "question": "## Task B-4.2.\n\nHow many complex numbers $z$ are there such that $|z|=1$ and the difference $z^{5!}-z^{4!}$ is a real number?",
        "answer": "240",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87425,
        "difficulty": null
      },
      {
        "id": "7b49843a4682ac262a2dd7c25e2ce343157f896479547eb53e58e6e8684deae9",
        "domain": "numina",
        "question": "[Area of a Triangle (using two sides and the angle between them)]\n\nA certain triangle can be cut out from a paper strip of unit width, but it cannot be cut out from any strip of lesser width. What area can this triangle have?\n\n#",
        "answer": "\\frac{1}{\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59318,
        "difficulty": null
      },
      {
        "id": "790dda7c4dfcef573297173e644757da86006d0ef4aeae02a010a6569614e681",
        "domain": "numina",
        "question": "2 [Rectangles and Squares. Properties and Characteristics]\n\nAuto: Dorricheno $C$.\n\nWhich are more numerous: rectangles with integer sides and a perimeter of 1996, or rectangles with integer sides and a perimeter of 1998?\n\n(Rectangles $a \\times b$ and $b \\times a$ are considered the same.)",
        "answer": "499",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 57246,
        "difficulty": null
      },
      {
        "id": "0db8eba46571de7684f474ba5b1a13eebfa551a8bd6b82fe1055b373ad17d1dc",
        "domain": "numina",
        "question": "4. In a regular 2017-gon, all diagonals are drawn. Petya randomly selects some $\\mathrm{N}$ diagonals. What is the smallest $N$ such that among the selected diagonals, there are guaranteed to be two of the same length?",
        "answer": "1008",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5532,
        "difficulty": null
      },
      {
        "id": "df1221d8eb95252a1336ff096a2ecd34c797514eb75f5bd3d5554a33d027f777",
        "domain": "numina",
        "question": "## 17. ILLUSTRATIVE EXERCISE\n\nA positive integer is called triangular if it can be written in the form $\\frac{n(n+1)}{2}$ for some positive integer $n$. How many pairs $(a, b)$ of triangular numbers are there such that $b-a=2007$? (It is recalled that 223 is a prime number).\n\nSOLUTION",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 181375,
        "difficulty": null
      },
      {
        "id": "07a008bf7635c688d7b3763ddc30bed5500920a511302f2174a9a3cb6cc2e40d",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the parametric equations.\n\n$$\n\\begin{aligned}\n& \\left\\{\\begin{array}{l}\nx=2(t-\\sin t) \\\\\ny=2(1-\\cos t)\n\\end{array}\\right. \\\\\n& 0 \\leq t \\leq \\frac{\\pi}{2}\n\\end{aligned}\n$$",
        "answer": "4(2-\\sqrt{2})",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47549,
        "difficulty": null
      },
      {
        "id": "a98b371a111b2b9f27487653db912169842fe15f6832e8949865da2df31c0015",
        "domain": "numina",
        "question": "1. Inside a cylinder with a base radius of 6, there are two spheres with radii also of 6, and the distance between their centers is 13. If a plane is made to be tangent to these two spheres and intersects the cylindrical surface to form an ellipse, then the sum of the lengths of the major and minor axes of this ellipse is $\\qquad$",
        "answer": "25",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 109584,
        "difficulty": null
      },
      {
        "id": "b1222c8019d249bb893ddda7f02a9b59c19f4fe82f5a99dc4687d707ccd93409",
        "domain": "numina",
        "question": "18 Let $S=1+\\frac{1}{\\sqrt{2}}+\\frac{1}{\\sqrt{3}}+\\cdots+\\frac{1}{\\sqrt{n}}$, to make $[S]=1994$, what should the value of $n$ be? ([S] represents the greatest integer not exceeding $S$).",
        "answer": "995006",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92393,
        "difficulty": null
      },
      {
        "id": "527225339c2036c03d7ac30803b0270029015a19dd55cc58e0e67382fe9b10b2",
        "domain": "numina",
        "question": "Kite $A B C D$ is symmetric with respect to the diagonal $A C$. The length of $A C$ is $12 \\mathrm{~cm}$, the length of $B C$ is $6 \\mathrm{~cm}$, and the internal angle at vertex $B$ is a right angle. Points $E$, $F$ are given on sides $A B, A D$ such that triangle $E C F$ is equilateral.\n\nDetermine the length of segment $E F$.\n\n(K. Pazourek)",
        "answer": "4\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76748,
        "difficulty": null
      },
      {
        "id": "b1ccac8742edca6bbe5a158de22f0d34e203ba195a373de97b9660f66f6c3979",
        "domain": "numina",
        "question": "6-1. The physical education teacher lined up the class so that everyone was facing him. To the right of Kolya, there are 12 people, to the left of Sasha - 20 people, and to the right of him - 8 people. How many people are standing to the left of Kolya?",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12352,
        "difficulty": null
      },
      {
        "id": "86c592f7b8da9e76a5b8930351891d74f37b888cd4770919f6a1fc9754800ff1",
        "domain": "numina",
        "question": "I took a classic black and white chessboard, which was made up of $8 \\times 8$ square tiles with sides of $3 \\mathrm{~cm}$. I rearranged the tiles within the given frame so that one black rectangle, one black square, and one contiguous white shape were formed. The individual tiles still touched each other with their entire sides after rearrangement. The black shapes did not touch (not even at a corner), and each of them had at least one side in common with the edge of the chessboard.\n\nDetermine the maximum possible perimeter of the white shape and draw how it might look in such a case.\n\n(M. Mach)\n\nHint. What dimensions could the square have?",
        "answer": "168",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76820,
        "difficulty": null
      },
      {
        "id": "84236e1cb0de2d4b0eaa36ca4e27835ca2d6ff19b52f75b9df62345c10fff132",
        "domain": "numina",
        "question": "One, Fill-in-the-blank Questions (8 points per question, total 64 points)\n1. Given the three sides of a triangle are consecutive natural numbers. If the largest angle is twice the smallest angle, then the perimeter of the triangle is $\\qquad$",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97958,
        "difficulty": null
      },
      {
        "id": "62049a009d7e7e7273138ac77f758bef6aafe4b4d994c84cc9b2e7cfdbfe851c",
        "domain": "numina",
        "question": "In an article about combinatorics, to determine the number of permutations, we essentially used an arrangement of notebooks that differs from the otherwise commonly used arrangement. In this arrangement, the rank of two notebooks is compared by sequentially checking how many lower-ranked elements are preceded by the second, third, fourth, etc., ranked elements. The notebook is of higher rank in which, proceeding in the above order, we first find an element that precedes more lower-ranked elements.\n\nFor example, comparing the groups 4231 and 3214: the 2 in both groups precedes one element, the 3 in the first group precedes one element, and in the second group, it precedes two elements, so in this special arrangement, the second group is of higher rank. (In the article, we saw that the situation is reversed in the usual arrangement.)\n\na) Write down all permutations of the elements 1, 2, 3, 4 in this special arrangement!\n\nb) What is the rank of the permutation 5341726 of the elements 1, 2, 3, 4, 5, 6, 7:\n\nI. in this special arrangement?\n\nII. in the usual arrangement?",
        "answer": "2271",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 172533,
        "difficulty": null
      },
      {
        "id": "95bb54e32d52b92be6dca1d6d76ea7feef554c80720f8c383c18d0d3b831ab35",
        "domain": "numina",
        "question": "5. Tangram is a familiar puzzle toy. Now, please use the cards provided to make a set of tangrams as shown in Figure 1, and take the four pieces numbered 1 to 4 to make a four-piece tangram.\n(1) Use the four pieces of the four-piece tangram to form the step shape shown in Figure 2. Draw the assembly method directly on Figure 2 with thick lines.\n(2) Figure 3 shows a four-level step of a cube, where the length, width, and height of each step are equal. It is known that the height $\\mathrm{AD}=h=3$ cm, the width $\\mathrm{DE}=b=1$ cm, and the length $\\mathrm{AC}=a=\\mathbf{8}$ cm. A clever mouse crawls from point A to point B along the surface of the steps (assuming it can crawl vertically on vertical surfaces), and it takes the shortest path. There is also a smart mousetrap that can be placed at any point on DE, FG, or HI. If it is placed on DE, then its movement path must be perpendicular to DE. Similarly, if it is placed on FG or HI, then its movement path must be perpendicular to FG or HI. It is known that the mouse and the smart mousetrap start at the same time, and the speed of the mouse $v=17$ cm/s. Prove that to catch the mouse exactly, the speed of the smart mousetrap is independent of its placement position, and find its speed.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137297,
        "difficulty": null
      },
      {
        "id": "3192787eec0c48cedb169a4d84e9a7081d6fb3e9ec7e7460fd4540d448ea8088",
        "domain": "numina",
        "question": "15. Given that $\\alpha$ is an acute angle, find the minimum value of $\\left(1+\\frac{1}{\\sin \\alpha}\\right)\\left(1+\\frac{1}{\\cos \\alpha}\\right)$.",
        "answer": "3+2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109849,
        "difficulty": null
      },
      {
        "id": "030630937f863581670a9da90b3a9ef5cbb79371c41aca8c582b6d793ef22c93",
        "domain": "numina",
        "question": "## Task 1\n\nThree pioneers collected a total of $139 \\mathrm{~kg}$ of scrap. The first pioneer brought $36 \\mathrm{~kg}$ to the collection point, and the second brought twice as much.\n\nHow many kilograms of scrap did the third pioneer collect?",
        "answer": "31",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186730,
        "difficulty": null
      },
      {
        "id": "b2f3a6dead2c4cda95583515d9a8359b75584b9a4cd725df42eb01850be62f5d",
        "domain": "numina",
        "question": "6. Let the non-empty set $A \\subseteq\\{1,2,3,4,5,6,7\\}$, when $a \\in A$ it must also have $8-a \\in A$, there are $\\qquad$ such $A$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "15",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 143918,
        "difficulty": null
      },
      {
        "id": "5d41ec357f0f11c8e827833a1429406135843ec7debed48029645793e8e078d4",
        "domain": "numina",
        "question": "16. (3 points) $A, B, C, D$ four boxes contain $8, 6, 3, 1$ balls respectively. The first child finds the box with the fewest balls, then takes one ball from each of the other boxes and puts it into this box; the second child also finds the box with the fewest balls, and then also takes one ball from each of the other boxes and puts it into this box, ..., after the 50th child has finished, the number of balls in box $A$ is $\\qquad$ .",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 93161,
        "difficulty": null
      },
      {
        "id": "865a0480e48a34382500c9b5f8582113522125c47d037de4649eef3f612a5f23",
        "domain": "numina",
        "question": "8. In how many different ways can integers $a, b, c \\in [1,100]$ be chosen so that the points with coordinates $A(-1, a), B(0, b)$, and $C(1, c)$ form a right triangle?",
        "answer": "974",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5985,
        "difficulty": null
      },
      {
        "id": "1aaf16a761a7f959e7cd417559d907a00fbeca6a8a5a65d0582fcb12a453f835",
        "domain": "numina",
        "question": "6. (8 points) By expanding the expression $(1+\\sqrt{5})^{209}$ using the binomial theorem, we obtain terms of the form $C_{209}^{k}(\\sqrt{5})^{k}$. Find the value of $k$ for which such a term attains its maximum value.",
        "answer": "145",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19422,
        "difficulty": null
      },
      {
        "id": "d63f2320e51b9f22a859216514c87a227eaff933ca86e235090fdc12e9e5c41b",
        "domain": "numina",
        "question": "The rectangle $A B C D$ is divided into five equal rectangles. If the perimeter of $A B C D$ is $20 \\mathrm{~cm}$, determine its area.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_80f7f1185898b39e7876g-04.jpg?height=414&width=499&top_left_y=2032&top_left_x=687)\n\n#",
        "answer": "\\frac{3000}{121}^2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73392,
        "difficulty": null
      },
      {
        "id": "d32432a5cfb64ea74f6cc67c46e7d3bd8666f777acc8e8de96291cf021942914",
        "domain": "numina",
        "question": "9. Half-planes $\\alpha_{1}, \\alpha_{2}, \\ldots, \\alpha_{n}$ have a common edge $l$. What is the sum of the dihedral angles $\\alpha_{1} \\widehat{l \\alpha_{2}}, \\alpha_{2} l \\alpha_{3}, \\ldots, \\alpha_{n-1} l \\alpha_{n}$, $\\widehat{\\alpha_{n} l} l \\alpha_{1}$, which together fill the entire space?",
        "answer": "2\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44401,
        "difficulty": null
      },
      {
        "id": "7f6df669e8279031629e77e7bf38008a7ff449ac167ebcf424409c36c9ba5158",
        "domain": "numina",
        "question": "I4.2 In Figure $1, \\triangle B A C$ is a right-angled triangle, $A B=A C=m \\mathrm{~cm}$. Suppose that the circle with diameter $A B$ intersects the line $B C$ at $D$, and the total area of the shaded region is $n \\mathrm{~cm}^{2}$. Find the value of $n$.",
        "answer": "\\frac{9}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153158,
        "difficulty": null
      },
      {
        "id": "8e31573affbf892453ae309762f5170829c1730829f8b4048c3e2207656c6409",
        "domain": "numina",
        "question": "We call a phone number $d_{1} d_{2} d_{3}-d_{4} d_{5} d_{6} d_{7}$ cool if the number $d_{1} d_{2} d_{3}$ is equal to $d_{4} d_{5} d_{6}$ or to $d_{5} d_{6} d_{7}$. For example, $234-2347$ is a cool phone number. Assume that each $d_{i}$ can be any digit from 0 to 9. How many cool phone numbers exist? \n\n#",
        "answer": "19990",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74016,
        "difficulty": null
      },
      {
        "id": "74c9f09c7a6633482311631cdb199745c00fcc38376e302a98c532c6b83cea67",
        "domain": "numina",
        "question": "11. It is necessary to clearly follow the requirements for the volume of work if it is specified in the assignment.\n\n## Task 1 (Maximum 12 points)\n\nLet there be 5 banks in the country Alpha. The probability that a bank will close is the same for all banks and is equal to 0.05. Banks close independently of each other. At some point, a crisis occurred in the country Alpha, and the probability that a bank will close became 0.25 (it remains the same for all banks, and banks still close independently of each other). Find how much, in absolute terms, the probability that at least one bank will close has changed. Round your answer to the nearest hundredth.",
        "answer": "0.54",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7418,
        "difficulty": null
      },
      {
        "id": "8317d1895dd50624c24ca752742e542aa27b0fc079e9a2c1d457c7b5f04a8799",
        "domain": "numina",
        "question": "Problem 8.2. The numbers from 1 to 9 were placed in the cells of a $3 \\times 3$ table such that the sum of the numbers on one diagonal is 7, and on the other diagonal is 21. What is the sum of the numbers in the five shaded cells?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_adefdf2067937cb67fc7g-28.jpg?height=416&width=428&top_left_y=105&top_left_x=513)",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18163,
        "difficulty": null
      },
      {
        "id": "df739e499bc1eaf6e1aeeaa0a92e160bcc83b6e2abaf8331d611e47d70baae90",
        "domain": "numina",
        "question": "5. Let $a^{b}=\\frac{1}{8}$. What is the value of $a^{-3 b}$ ?",
        "answer": "512",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156393,
        "difficulty": null
      },
      {
        "id": "20d09f966375c3a65a7bba4868513c386c9a133095a8827200574c458adb44fa",
        "domain": "numina",
        "question": "B2 This month, I spent 26 days exercising for 20 minutes or more, 24 days exercising 40 minutes or more, and 4 days of exercising 2 hours exactly. I never exercise for less than 20 minutes or for more than 2 hours.\nWhat is the minimum number of hours I could have exercised this month?",
        "answer": "22",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118658,
        "difficulty": null
      },
      {
        "id": "43ed058306ed066635234246ee92bb79152fb6cb3242a3e0a3052d22d9f0f424",
        "domain": "numina",
        "question": "1. Given the complex number $z$ satisfies $\\frac{z-\\mathrm{i}}{z-1}$ is a pure imaginary number (where $\\mathrm{i}$ is the imaginary unit), then the maximum value of $|z|$ is",
        "answer": "\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96701,
        "difficulty": null
      },
      {
        "id": "49fafb0ed5b8ec03775c8c2a1b11a588c0928d9d9615ee10ecc43974c1afab7f",
        "domain": "numina",
        "question": "## Task B-4.3.\n\nA circle with center at point $S$ is inscribed in triangle $A B C$. The circle touches the sides of the triangle at points $D \\in \\overline{B C}, E \\in \\overline{A C}$ and $F \\in \\overline{A B}$. If $|A E|=3 \\mathrm{~cm},|C S|=2 \\sqrt{7} \\mathrm{~cm}$, and the measure of the angle at vertex $A$ is $60^{\\circ}$, calculate the area of triangle $A B C$.",
        "answer": "10\\sqrt{3}^2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88773,
        "difficulty": null
      },
      {
        "id": "a0a5ab94aaf22048ed2059fc9a51c77c3e965b4a71e0e9605ec5a8ccde614ec6",
        "domain": "numina",
        "question": "81. Select several numbers from $1,2,3,4,5,6$ so that the sum of the selected numbers is a multiple of 3 but not a multiple of 5. There are $\\qquad$ different ways to do this.",
        "answer": "19",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119120,
        "difficulty": null
      },
      {
        "id": "9050db3c7d45cf607125b8a89238af8d80edb1cda6a0e79b368f140291814d78",
        "domain": "numina",
        "question": "2. Determine the largest value of $x$ for which\n$$\n\\left|x^{2}-4 x-39601\\right| \\geq\\left|x^{2}+4 x-39601\\right|\n$$",
        "answer": "199",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160627,
        "difficulty": null
      },
      {
        "id": "f2e7206a6f5a815d1f2f9c2ea284bc88624b0603d5785b91082ba4df6954e89e",
        "domain": "numina",
        "question": "2. Laura has 2010 lamps and 2010 switches in front of her, with different switches controlling different lamps. She wants to find the correspondence between the switches and the lamps. For this, Charlie operates the switches. Each time Charlie presses some switches, and the number of lamps that light up is the same as the number of switches pressed (Note: The switches return to their original state after each operation). Questions:\n(1) What is the maximum number of different operations Charlie can perform so that Laura can accurately determine the correspondence between the switches and the lamps?\n(2) If Laura operates the switches herself, what is the minimum number of operations she needs to find the correspondence between the switches and the lamps?",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 137798,
        "difficulty": null
      },
      {
        "id": "78ad551b88351cee3834d6256c5665c8fd586dc32c7c455d47d737ad09448608",
        "domain": "numina",
        "question": "1. (5 points) Find the value of $n$ for which the following equality holds:\n\n$$\n\\frac{1}{1+\\sqrt{2}}+\\frac{1}{\\sqrt{2}+\\sqrt{3}}+\\frac{1}{\\sqrt{3}+\\sqrt{4}}+\\ldots+\\frac{1}{\\sqrt{n}+\\sqrt{n+1}}=2011\n$$",
        "answer": "4048143",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19444,
        "difficulty": null
      },
      {
        "id": "a87aa7d3af1ce24266ee2bf7558b0183c858625e5b70fb8d475edc8d32801d3d",
        "domain": "numina",
        "question": "7. (36th Austrian Mathematical Olympiad) Let $a$ be an integer, and $|a| \\leqslant 2005$. Find the number of $a$ such that the system of equations $\\left\\{\\begin{array}{l}x^{2}=y+a, \\\\ y^{2}=x+a\\end{array}\\right.$ has integer solutions.",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 106217,
        "difficulty": null
      },
      {
        "id": "69007fcbd1323e4255a5ae3ad6be4ade8cbc3e8433f7a79720107cfbb77a61f6",
        "domain": "numina",
        "question": "IS. $3 W$ and $F$ are two integers which are greater than 20. If the product of $W$ and $F$ is $b$ and the sum of $W$ and $F$ is $c$, find the value of $c$.",
        "answer": "57",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152854,
        "difficulty": null
      },
      {
        "id": "c16a97c5b255b464f4fa196363248ccfcf1ec61bae9cc3d7866deccb8824b783",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{1-2+3-4+\\ldots+(2 n-1)-2 n}{\\sqrt[3]{n^{3}+2 n+2}}\n$$",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47059,
        "difficulty": null
      },
      {
        "id": "906ec7628a2813e9088756fe5fffd2917f0e725f34d067a44c0407f96f01b51a",
        "domain": "numina",
        "question": "Can $n!$ end with exactly five zeros?\n\n#",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 71495,
        "difficulty": null
      },
      {
        "id": "4913d3b1033acb88bbae9d86e9fd44bdd71b5962132f71c9186bba3b70c59c65",
        "domain": "numina",
        "question": "1. The picture shows several circles connected by segments. Sasha chooses a natural number \\( n \\) and places different natural numbers in the circles so that for all these numbers the following property holds:\n\nif numbers \\( a \\) and \\( b \\) are not connected by a segment, then the sum \\( a + b \\) must be coprime with \\( n \\), while if they are connected, then the numbers \\( a + b \\) and \\( n \\) must have a common natural divisor greater than 1. For what smallest \\( n \\) does such an arrangement exist?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0b0e793e58d3be992fedg-01.jpg?height=274&width=277&top_left_y=800&top_left_x=907)",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 8375,
        "difficulty": null
      },
      {
        "id": "22370c453bc35c108ac39cb509159d57b3d4421dea41c34fa6bea300b3e545e7",
        "domain": "numina",
        "question": "8. On a birthday card printed with April 29, a child added two positive integers $x$ and $y$ between 4 and 29, resulting in a five-digit number $\\overline{4 x y 29}$. It turns out that this number is exactly the square of the positive integer $T$ corresponding to their birthday: $\\overline{4 x y 29}=T^{2}$. Then the number $T$ corresponding to the child's birthday is $T=$ $\\qquad$",
        "answer": "223",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 119028,
        "difficulty": null
      },
      {
        "id": "7eedb84b159ee92c38670aa5d2259f321e0dce1476453dfef7b63483c0fabc5e",
        "domain": "numina",
        "question": "L OM - I - Task 5\n\nFind all pairs of positive integers $ x $, $ y $ satisfying the equation $ y^x = x^{50} $.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 1080,
        "difficulty": null
      },
      {
        "id": "b634d8a62ce55d78df57eca78a9c736740fc86e5585387333f3c16e40f647573",
        "domain": "numina",
        "question": "13. There are two bags, $A$ and $B$. Bag $A$ contains 6 cards, one of which is labeled 0, two are labeled 1, and three are labeled 2; Bag $B$ contains 7 cards, four of which are labeled 0, one is labeled 1, and two are labeled 2. One card is drawn from bag $A$ and two cards are drawn from bag $B$, making a total of three cards. Find\n(1) the probability that all three cards drawn are 0;\n(2) the probability that the product of the numbers on the three cards is 4.",
        "answer": "\\frac{4}{63}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 139864,
        "difficulty": null
      },
      {
        "id": "d90dec084f9d4884abc1f2837909fcb45c20e7d4eb4678d40c32bbccc4e9b68d",
        "domain": "numina",
        "question": "4. (8 points) There are four people, A, B, C, and D. B owes A 1 yuan, C owes B 2 yuan, D owes C 3 yuan, and A owes D 4 yuan. To settle all debts among them, A needs to pay $\\qquad$ yuan.\n\n保留源文本的换行和格式，翻译结果如下：\n\n4. (8 points) There are four people, A, B, C, and D. B owes A 1 yuan, C owes B 2 yuan, D owes C 3 yuan, and A owes D 4 yuan. To settle all debts among them, A needs to pay $\\qquad$ yuan.",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 144442,
        "difficulty": null
      },
      {
        "id": "9971ab79fe7e3d25b6019bdccbf5c75083a1c9b81a15fc34125af0ba0ebbc3f3",
        "domain": "numina",
        "question": "Problem 5. If Mare buys 20 eggs from the market, 30 denars will remain from the amount she brought, and to buy 30 eggs, 20 denars are missing. How many denars did Mare bring to the market?",
        "answer": "130",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188856,
        "difficulty": null
      },
      {
        "id": "79c9148b75650094f3e8046b1e321c772f5b2f2177e506732a3779ee90760f4e",
        "domain": "numina",
        "question": "## Task 1 - 080811\n\nThe figure shows a five-pointed star, where the points $A, B, C, D, E$ are the vertices of a regular pentagon.\n\nDetermine the size of the angle $\\angle A C E$!\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0621.jpg?height=354&width=352&top_left_y=448&top_left_x=1366)",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 184005,
        "difficulty": null
      },
      {
        "id": "f5618f0935eee0a2272add0863280df7d6d9a0859dc710f356c34a93f129dfd3",
        "domain": "numina",
        "question": "10.348. A convex quadrilateral is divided by its diagonals into four triangles; the areas of three of them are 10, 20, and $30 \\mathrm{~cm}^{2}$, and each is less than the area of the fourth triangle. Find the area of the given quadrilateral.",
        "answer": "120^2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 48953,
        "difficulty": null
      },
      {
        "id": "a7bb6ef497346468c5f3d08eedfbcc949ed481029ac2898b8642c214b9674d12",
        "domain": "numina",
        "question": "4. Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=\\sqrt{x^{2}+1}+\\sqrt{(x-12)^{2}+16}$ is",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101214,
        "difficulty": null
      },
      {
        "id": "8bec68382a88d53a21f6d0f44beafb3e9a607faf432acfb3caed13bc1aa7d490",
        "domain": "numina",
        "question": "7. Let the pair of positive integers $(x, y)$ satisfy $\\frac{1}{\\sqrt{x}}+\\frac{1}{\\sqrt{y}}=\\frac{1}{\\sqrt{20}}$. Then $xy$ has $\\qquad$ different possible values.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117770,
        "difficulty": null
      },
      {
        "id": "7995c956970c6677ddeeaa89e8e4ec8f696d34c35253d2e60002d4440b1666c4",
        "domain": "numina",
        "question": "11-8. In a caravan, there are 100 camels, both one-humped and two-humped, and there are at least one of each. If you take any 62 camels, they will have at least half of the total number of humps in the caravan. Let $N$ be the number of two-humped camels. How many values (in the range from 1 to 99) can $N$ take?",
        "answer": "72",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12399,
        "difficulty": null
      },
      {
        "id": "e99a70da3cd660a1bab3633613ad9d7faf6fdcd3fa08e740098f228241a0dde1",
        "domain": "numina",
        "question": "## Task 24/84\n\nDetermine all four-digit natural numbers n (in the decimal system) that satisfy the following conditions:\n\n1. The cross sum is 23.\n2. The alternating cross sum is -5.\n3. The cross product is 360.\n4. The sum of the digits in the first and third positions is equal to the digit in the second position.",
        "answer": "19858915",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182744,
        "difficulty": null
      },
      {
        "id": "178d1d32c6100e394d7b11542ef37b157e5b8ce8808f130892654bf99ef58599",
        "domain": "numina",
        "question": "20. (6 points) A group of monkeys needs to transport peaches from location $A$ to location $B$. Every 3 minutes, a monkey starts from $A$ and heads to $B$, and the entire journey takes 12 minutes. A rabbit starts running from $B$ to $A$ at the same time a monkey arrives at $B$. On its way, the rabbit encounters 5 monkeys coming towards it, and continues to $A$, where it meets another monkey just starting from $A$. If the rabbit's running speed is 3 kilometers/hour, then the distance between $A$ and $B$ is $\\qquad$",
        "answer": "300",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 92895,
        "difficulty": null
      },
      {
        "id": "13a76a115e52e2e99152c1323a6e8fc35e2e0ed8ea0c0fd2329d8c258efcfda2",
        "domain": "numina",
        "question": "In a table tennis tournament, $n$ Asian and $2 n$ European players participated. The number of matches won by the Europeans is 5/7 of the number of matches won by the Asians. How many players participated in the tournament?",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 165762,
        "difficulty": null
      },
      {
        "id": "6b2715eb605289d2de43e29b4629884df275788e96d5e031dbfc83b0b1b068f7",
        "domain": "numina",
        "question": "## Task A-1.2.\n\nLet $a$ and $b$ be positive real numbers such that\n\n$$\n\\frac{a}{b}+\\frac{b}{a}=3 \\quad \\text { and } \\quad \\frac{a^{2}}{b}+\\frac{b^{2}}{a}=10\n$$\n\nDetermine $\\frac{1}{a}+\\frac{1}{b}$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88807,
        "difficulty": null
      },
      {
        "id": "4bdd3383d5d876ecb2de37f381e2509dc375af027b7d2cb55ca2bf790dfd174c",
        "domain": "numina",
        "question": "5. (3 points) Three circles with radii 1, 1, and \\(2 \\sqrt{\\frac{13-6 \\sqrt{3}}{13}}\\) are arranged such that the triangle formed by their centers is equilateral with a side length of \\(\\sqrt{3}\\). Find the radius of the circumcircle of the triangle, each vertex of which is the point of intersection of two of these circles, farthest from the center of the third circle.\n\nAnswer: \\(4 \\sqrt{3}-6\\)",
        "answer": "4\\sqrt{3}-6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20992,
        "difficulty": null
      },
      {
        "id": "2638d64b77e467e7c1b77a2dcbcaf25a6033f56017b9fc45dd63ef09d21457fe",
        "domain": "numina",
        "question": "6. Points $E$ and $F$ are the midpoints of sides $\\overline{B C}$ and $\\overline{C D}$ of rectangle $A B C D$. What is the area of triangle $A E F$ if the area of rectangle $44 \\mathrm{~cm}^{2}$?",
        "answer": "16.5^2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86743,
        "difficulty": null
      },
      {
        "id": "8a5157783855cb53d03ca099fe02e274a03cabb91b9bedf9529bc9fba730f2e1",
        "domain": "numina",
        "question": "9. (16 points) Let\n$$\n\\begin{aligned}\nP(z)= & z^{4}-(6 \\mathrm{i}+6) z^{3}+24 \\mathrm{i}^{2}- \\\\\n& (18 \\mathrm{i}-18) z-13 .\n\\end{aligned}\n$$\n\nFind the area of the convex quadrilateral formed by the four points in the complex plane corresponding to the four roots of $P(z)=0$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136748,
        "difficulty": null
      },
      {
        "id": "3aca20c063ab56abaaeafe13f4e479d367439ad9e7890046f0616f47a73e321b",
        "domain": "numina",
        "question": "7. The dog is chasing a rabbit that is 90 of its own jumps ahead of it. While the rabbit jumps 6 times, (20) the dog jumps only 5 times, and two dog jumps are as long as three rabbit jumps. How many jumps will the rabbit make until the dog catches it?",
        "answer": "360",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88079,
        "difficulty": null
      },
      {
        "id": "6109dd537acbe21f8810b6c3b7462790b5faf3e9978b48ab60fe60893ae9bdc3",
        "domain": "numina",
        "question": "## Task 2 - 090612\n\nDuring the holidays, Klaus was in the countryside. From his observations, the following joke problem emerged:\n\n$$\n1 \\frac{1}{2} \\text { chickens lay in } 1 \\frac{1}{2} \\text { days } 1 \\frac{1}{2} \\text { eggs }\n$$\n\nDetermine the total number of eggs that 7 chickens would lay in 6 days at the same laying rate!",
        "answer": "28",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 183235,
        "difficulty": null
      },
      {
        "id": "3e008edd3ffe202bf0896ed1c42233cf410ae953bca049eccdbd589b43d2f779",
        "domain": "numina",
        "question": "7、A book's page numbers are consecutive natural numbers, $1,2,3, \\cdots$, when these page numbers are added together, one page number is added twice, resulting in the incorrect sum of 1997, then the page number that was added twice is $\\qquad$",
        "answer": "44",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 111987,
        "difficulty": null
      },
      {
        "id": "31e460646691f6936c06f9ab7f37fc9404819c2c55c870a401f5b4d69bffefbf",
        "domain": "numina",
        "question": "Question 9: Given that $f(x)$ is an odd function defined on $R$, and when $x<0$, $f(x)$ is monotonically increasing, $f(-1)=0$. Let $\\varphi(x)=\\sin ^{2} x+m \\cdot \\cos x-2 m$, set $M=\\left\\{m \\mid\\right.$ for $\\left.\\forall x \\in\\left[0, \\frac{\\pi}{2}\\right], \\varphi(x)<0\\right\\}, N=\\{m \\mid$ for $\\left.\\forall \\mathrm{x} \\in\\left[0, \\frac{\\pi}{2}\\right], \\mathrm{f}(\\varphi(\\mathrm{x}))<0\\right\\}$, find $\\mathrm{M} \\cap \\mathrm{N}$.",
        "answer": "4-2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91850,
        "difficulty": null
      },
      {
        "id": "e0f67b50d57bcc30c3fbb43a7765e1a93ff24ef761d346c68e3ea4b85ced5eb2",
        "domain": "numina",
        "question": "3. In a cube with edge length 1, construct a large inscribed sphere $O_{1}$, and then construct a smaller sphere $O_{2}$ inside the cube, such that it is externally tangent to the larger sphere and also tangent to three faces of the cube. Then the surface area of sphere $O_{2}$ is $\\qquad$ .",
        "answer": "(7-4\\sqrt{3})\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 110516,
        "difficulty": null
      },
      {
        "id": "d83ef7804688ad97073367159e901570284ada450937709711a99ede76c44f0f",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nGray has $t$ dollars consisting of $\\$ 1$ and $\\$ 2$ coins. If she has the same number of $\\$ 1$ and $\\$ 2$ coins, how many $\\$ 1$ coins does she have?\n\n##",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194116,
        "difficulty": null
      },
      {
        "id": "0af58ab216a1e15ce9c69331c8167dc0e4fbe65623b28bd7f2471e8446660c9c",
        "domain": "numina",
        "question": "B1. Every three-digit number can be written in reverse. 176 can be reversed to 671.\n\nDetermine all three-digit numbers that are equal to 3 times their reversed plus the sum of their digits.\n\nIn formula: $\\overline{d c}=3 \\times \\overline{d a}+(a+b+c)$",
        "answer": "441882",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85840,
        "difficulty": null
      },
      {
        "id": "25f7fa60b9ab713a2751e7955b0dccfdee220917aa98a49e2ae7c0a135c84620",
        "domain": "numina",
        "question": "15th Putnam 1955 Problem A6 For what positive integers n does the polynomial p(x) ≡ x n + (2 + x) n + (2 - x) n have a rational root. Solution",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 149262,
        "difficulty": null
      },
      {
        "id": "9364af21927570d2f2efdb6559aff2dc6bf560e6f341fb0c6d762312fd043b8c",
        "domain": "numina",
        "question": "9. The base of the pyramid $T A B C D$ is an isosceles trapezoid $A B C D$, the length of the smaller base $B C$ of which is $\\sqrt{3}$. The ratio of the areas of the parts of the trapezoid $A B C D$, into which it is divided by the midline, is $5: 7$. All lateral faces of the pyramid $T A B C D$ are inclined to the base plane at an angle of $30^{\\circ}$. The plane $A K N$, where points $K$ and $N$ are the midpoints of edges $T B$ and $T C$ respectively, divides the pyramid into two parts. Find the volume of the larger part.\n\n$(16$ points)",
        "answer": "0.875",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21761,
        "difficulty": null
      },
      {
        "id": "c515d337828670ec7f6980f2cb2e4c184e895588d6f7901f3c82c3d07e674e8e",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 1} \\frac{x^{2}-1}{\\ln x}$",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47366,
        "difficulty": null
      },
      {
        "id": "66066fdc5f3a0c9f1184c6cfb981926d2ed4bac8fb425c84ed1a45de5ea03508",
        "domain": "numina",
        "question": "Sure, here is the translated text:\n\n```\nI. Fill-in-the-Blanks (8 points per question, total 64 points)\n1. Given the sets\n$$\nA=\\{x \\mid 5 x-a \\leqslant 0\\}, B=\\{x \\mid 6 x-b>0\\}\n$$\n\nwhere $a, b \\in \\mathbf{N}_{+}$.\nIf $A \\cap B \\cap \\mathbf{N}=\\{2,3,4\\}$, then the number of integer pairs $(a, b)$ is . $\\qquad$\n```",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114977,
        "difficulty": null
      },
      {
        "id": "621053531c2cfbd1e330a3fc57fd7fcb08bf030c4bc062a6b4b28e382e644eb1",
        "domain": "numina",
        "question": "## Task 1 - 030511\n\nThe VEB Simson Suhl currently produces 235 microcars KR 50 daily. In 1958, production was only 42 microcars per day.\n\nHow many more microcars were produced in 1963 compared to 1958? Assume the number of working days in a year is 300.",
        "answer": "57900",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 182915,
        "difficulty": null
      },
      {
        "id": "e4caea01473f0ff29b6482e925e990b1c8ad629d0bd2263a90a1cf60c1fa7bf2",
        "domain": "numina",
        "question": "11. (20 points) As shown in Figure 1, for the ellipse $\\Gamma: \\frac{x^{2}}{4}+y^{2}=1$, two chords $A B$ and $C D$ satisfy $\\overrightarrow{D C}=2 \\overrightarrow{A B}$. Let the intersection of lines $A D$ and $B C$ be point $P$.\n(1) Find the maximum value of $|A B|+|C D|$; 360227734\n(2) If point $P$ lies on the parabola $y=\\frac{1}{4} x^{2}-3$, find the maximum area of quadrilateral $A B C D$.",
        "answer": "\\frac{(3\\sqrt{13}-6)\\sqrt{2+2\\sqrt{13}}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 104836,
        "difficulty": null
      },
      {
        "id": "2f55bec0968e6a042922ffae1678cd7d77073f59dd95fb64f863cce73b44116e",
        "domain": "numina",
        "question": "## Task 3 - 341213\n\nThe circles in the figure can be filled with real numbers such that the given numbers appear in the outer circles and that in each of the seven inner circles, the arithmetic mean of the six adjacent circles is placed.\n\nDetermine which number $x$ stands in the central circle.\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-2349.jpg?height=409&width=397&top_left_y=2257&top_left_x=1315)",
        "answer": "\\frac{20}{3}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186280,
        "difficulty": null
      },
      {
        "id": "1b0b95bbe92bc58fbc6fa4a9fe435d27784732746e303ff7fa1dad0069febc83",
        "domain": "numina",
        "question": "## 37. International Conference\n\nFour French, two representatives from the Republic of Ivory Coast, three English and four Swedes gathered at a conference to discuss forestry issues. The conference participants from the same country initially sat next to each other on a bench with 13 seats during the opening ceremony, and later, during the working sessions, at a round table with 13 seats.\n\nIn how many different ways can the conference participants sit next to each other in each of these two cases?",
        "answer": "165888",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 42015,
        "difficulty": null
      },
      {
        "id": "7342c4c1f1cc0f08c1663ae2006c508de6f334cdd96f7c06d063b2c0a962d7c0",
        "domain": "numina",
        "question": "4. A table tennis tournament proceeds according to the following rule. In each round, if the number of participants is odd, one is randomly selected to automatically advance to the next round. The remaining participants $\\mathrm{z}$ are paired by random draw. The players of each pair compete against each other, and the winner of each pair advances to the next round. Let $f(n)$ denote the number of rounds in a tournament with $n$ participants. (For example, $f(5)=3$.) Determine $f(2013)$ and find the smallest natural number $n$ for which $f(n)=f(2013)$.\n\n## 57th Mathematical Competition for High School Students in Slovenia\n\nCelje, April 20, 2013\n\n## Solutions to Problems and Scoring\n\n(April 21, 2013, $14: 53$ )",
        "answer": "1025",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 162945,
        "difficulty": null
      },
      {
        "id": "149ae6af8569eab2e4dbb8bbf46fc1484e583ce2c1bc2c50d6073e9ebdaec8ba",
        "domain": "numina",
        "question": "3. A balloon that blows up in the shape of a perfect cube is being blown up at a rate such that at time $t$ fortnights, it has surface area $6 t$ square furlongs. At how many cubic furlongs per fortnight is the air being pumped in when the surface area is 144 square furlongs?",
        "answer": "3\\sqrt{6}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 152064,
        "difficulty": null
      },
      {
        "id": "87661ba75e62c007ba3f703ce8e883ce83eb28b8e331659a6db4003534418147",
        "domain": "numina",
        "question": "7. Determine all natural numbers of the form $\\overline{a b c d e f}$ such that in their representation they have the digits 1, 2, 3, 4, 5, and 6, none of which repeat, and the following conditions are met:\n\n$\\overline{a b}$ is divisible by 2,\n\n$\\overline{a b c}$ is divisible by 3,\n\n$\\overline{a b c d}$ is divisible by 4,\n\n$\\overline{a b c d e}$ is divisible by 5,\n\n$\\overline{a b c d e f}$ is divisible by 6.\n\n## Ministry of Science and Education of the Republic of Croatia  Agency for Education and Upbringing  Croatian Mathematical Society\n\n## SCHOOL/CITY COMPETITION IN MATHEMATICS  January 26, 2023.  8th grade - elementary school\n\nIn addition to the final result, the procedure is also graded. To receive full points, it is necessary to find all solutions and confirm that there are no others, write down the procedure, and justify your conclusions. Problems 1. - 5. are worth six points each, and 6. and 7. are worth ten points each.\n\n## Problems worth 6 points:",
        "answer": "123654321654",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86828,
        "difficulty": null
      },
      {
        "id": "872183a93892f81cb4d60f50812625a6bca56ab358be8b1b39b8e0a9c9ea6ebf",
        "domain": "numina",
        "question": "4. Let the edge length of the regular tetrahedron $ABCD$ be 2. Then the sum of the areas of all sections of the circumsphere of this tetrahedron by planes that are equidistant from the four vertices of the tetrahedron is $\\qquad$",
        "answer": "\\frac{59\\pi}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 111481,
        "difficulty": null
      },
      {
        "id": "7bca84c5d381e28bf83247bc90615a5841f3a6165a7d2f17767aa696ea82e07e",
        "domain": "numina",
        "question": "8. Given the hyperbola $H: x^{2}-y^{2}=1$ and a point $M$ in the first quadrant on $H$, the line $l$ is tangent to $H$ at point $M$ and intersects the asymptotes of $H$ at points $P, Q$ (with $P$ in the first quadrant). If point $R$ is on the same asymptote as $Q$, then the minimum value of $\\overrightarrow{R P} \\cdot \\overrightarrow{R Q}$ is $\\qquad$",
        "answer": "-\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 146634,
        "difficulty": null
      },
      {
        "id": "ff10c9adfd9baa8fa7ca42034352652f752fae203e0f71a4a4ccb270bbfc34c2",
        "domain": "numina",
        "question": "G5 Inside the square $A B C D$, the equilateral triangle $\\triangle A B E$ is constructed. Let $M$ be an interior point of the triangle $\\triangle A B E$ such that $M B=\\sqrt{2}, M C=\\sqrt{6}, M D=\\sqrt{5}$ and $M E=\\sqrt{3}$. Find the area of the square $A B C D$.",
        "answer": "3+\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 169,
        "difficulty": null
      },
      {
        "id": "d81773e225dca8da20debc8a8fbff13bec7e16dc193a506d4f3a77046c61a6bb",
        "domain": "numina",
        "question": "Task B-4.4. Determine all values of the real parameter $p$ for which the equation $x^{4}-(3 p+2) x^{2}+p^{2}=0$ has four solutions that form four consecutive terms of an arithmetic sequence.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87237,
        "difficulty": null
      },
      {
        "id": "9b3f87c0331e562909f8bf30dd695bb248990e0ad16fae92c6d52b47258e2dd2",
        "domain": "numina",
        "question": "What is the probability that two interns were born on the same day (not necessarily in the same year)? Note: There are 62 interns.",
        "answer": "99.59095749",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 80872,
        "difficulty": null
      },
      {
        "id": "5b5ff3f5c21bd43e49aaa12bc56c53f36ebddc32f250098e1888aec6db932709",
        "domain": "numina",
        "question": "11.213 Calculate the volume of a triangular pyramid, in which two opposite edges are 4 and 12 cm, and each of the other edges is 7 cm.",
        "answer": "24^3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 52126,
        "difficulty": null
      },
      {
        "id": "d2a64e89e6b7c4029d2a3cae1ecdb606cab7ba8f36ba7b97482674eb4cd783b4",
        "domain": "numina",
        "question": "5. If four dice are thrown at random, the probability that the smallest number shown by these four dice is exactly 3 is\n$\\qquad$ (Supplied by Zhao Bin)",
        "answer": "\\frac{175}{1296}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117420,
        "difficulty": null
      },
      {
        "id": "b95df4ef0fe901867947670c39e53b4d71a22bf167063b579670b926f175be25",
        "domain": "numina",
        "question": "I4.4 In Figure 2, there are a square $A B C D$ with side length $(R-1) \\mathrm{cm}$ and an equilateral triangle $A E F$. ( $E$ and $F$ are points on $B C$ and $C D$ respectively). If the area of $\\triangle A E F$ is $(S-3) \\mathrm{cm}^{2}$, find the value of $S$.",
        "answer": "2\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153112,
        "difficulty": null
      },
      {
        "id": "e70f44bb6fc3d9b5b6d5c1c9a22186816d9b079cac1522a9849a6fb858ab30c2",
        "domain": "numina",
        "question": "19. Let $a, b, c, d$ be real numbers such that\n$$\n\\left\\{\\begin{array}{l}\na^{2}+b^{2}+2 a-4 b+4=0 \\\\\nc^{2}+d^{2}-4 c+4 d+4=0 .\n\\end{array}\\right.\n$$\n\nLet $m$ and $M$ be the minimum and the maximum values of $(a-c)^{2}+(b-d)^{2}$, respectively. What is $m \\times M$ ?",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160666,
        "difficulty": null
      },
      {
        "id": "477fbf17ad6c6223fcda4e45e5fd933206f8629ddf5b38c83b9a019303d5f9af",
        "domain": "numina",
        "question": "3. Find the last three digits of the non-zero natural number $n$, knowing that when $29 n$ is divided by 250, the remainder is 67, and when $23 n$ is divided by 200, the remainder is 29.\n\n(Gazeta Matematica)",
        "answer": "123",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 178779,
        "difficulty": null
      },
      {
        "id": "5aae180c56edf8c106bb27f361727e751aae4eab50d80c0478500f8dba946bd8",
        "domain": "numina",
        "question": "If $a, b$, and $c$ are positive real numbers with $2 a+4 b+8 c=16$, what is the largest possible value of $a b c$ ?",
        "answer": "\\frac{64}{27}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90447,
        "difficulty": null
      },
      {
        "id": "3eeb23cd14c85641842578acd06e8f44f834fadef4d4c0bd29df7a538702f971",
        "domain": "numina",
        "question": "6. A five-digit number abcde satisfies:\n$$\nac>d, dd, b>e \\text {. }\n$$\n\nFor example, 34201, 49412. If the digits of the number change with the position in a way similar to the monotonicity of a sine function over one period, then the five-digit number is said to conform to the \"sine rule.\" The number of five-digit numbers that conform to the sine rule is $\\qquad$.",
        "answer": "2892",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 126246,
        "difficulty": null
      },
      {
        "id": "2f689b112c2bb7f55560af21c9d805197e5bb95e2b0a62eeda2b4e75db5cec38",
        "domain": "numina",
        "question": "5. Let the even function $f(x)$ satisfy: $f(1)=2$, and when $x y \\neq 0$, $f\\left(\\sqrt{x^{2}+y^{2}}\\right)=\\frac{f(x) f(y)}{f(x)+f(y)}$. Then $f(5)=$ $\\qquad$.",
        "answer": "\\frac{2}{25}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120425,
        "difficulty": null
      },
      {
        "id": "8de093a580c1e84ba690f6fcca0a6c1b923c007333242bbe758f92e894f7f186",
        "domain": "numina",
        "question": "3. In $\\triangle A B C$, $\\angle C=90^{\\circ}, \\angle A$ and $\\angle B$ are bisected and intersect at point $P$, and $P E \\perp A B$ at point $E$. If $B C=2, A C=3$,\nthen $A E \\cdot E B=$ $\\qquad$",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 120510,
        "difficulty": null
      },
      {
        "id": "f3205d7568744ca534742d8cae67d9f15dd5d289b979c103de7535b6b8d1e2bd",
        "domain": "numina",
        "question": "\\section*{Task 2 - 011112}\n\nA steamboat travels from A to B in 3 hours and, with the same engine power, from B to A in \\(4 \\frac{1}{2}\\) hours. How long does a vessel propelled only by the current take for the journey from A to B?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 185535,
        "difficulty": null
      },
      {
        "id": "1660f977009c7307976d50fef0d3105a2f4c1a21461ae6174d88df91773e86c5",
        "domain": "numina",
        "question": "Evoikinov M.A.\n\nA pirate has five bags of coins, each containing 30 coins. He knows that one bag contains gold coins, another contains silver coins, a third contains bronze coins, and each of the two remaining bags contains an equal number of gold, silver, and bronze coins. You can simultaneously take any number of coins from any of the bags and see what kind of coins they are (coins are taken out only once). What is the smallest number of coins that need to be taken to definitely determine the contents of at least one bag?",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 53573,
        "difficulty": null
      },
      {
        "id": "887846c43570ed835e3d1e5afc8f0d9c9546c0a006649df32a49f30b77c0ed63",
        "domain": "numina",
        "question": "7. (15 points) The rules of a \"fun game\" are as follows: The player has four skills: Water, Fire, Wind, and Earth. Each skill requires a certain number of crystals to activate, and the specific effects of each skill are as follows:\nWater: Consumes 4 crystals, and simultaneously reduces the enemy's health by 4 points (if the enemy's health is less than 4 points, it is directly killed); Fire: Consumes 10 crystals, and simultaneously reduces the enemy's health by 11 points (if the enemy's health is less than 11 points, it is directly killed); Wind: Consumes 10 crystals, and simultaneously reduces the enemy's health by 5 points (if the enemy's health is less than 5 points, it is directly killed) and the next skill's crystal consumption is halved (for example: if the Water skill is used after the Wind skill, the Water skill only consumes 2 crystals);\nEarth: Consumes 18 crystals, and simultaneously halves the enemy's health (if the enemy's health is odd, add 1 first and then divide by 2): If the enemy's health is 120 points, then by reasonably selecting skills, at least $\\qquad$ crystals are needed to kill the enemy (the enemy is killed when its health is reduced to 0).",
        "answer": "63",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 137710,
        "difficulty": null
      },
      {
        "id": "ebeaccdef484e6359095062702e9da9c639cfa92f0fdd52ccead4b6840fd709e",
        "domain": "numina",
        "question": "14.1.11 * Try to append 3 digits after $523 \\cdots$ so that the resulting 6-digit number is divisible by $7$, $8$, and $9$.",
        "answer": "523656523152",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 128244,
        "difficulty": null
      },
      {
        "id": "7e1a031ccd5b9da7c5048b9d196a8fb1425056b9bb83f2b089cabee59522a732",
        "domain": "numina",
        "question": "52. Equal Distances. A few days ago, Professor Rackbrain stunned his students with the following puzzle:\n\n- When between three and four o'clock is the minute hand at the same distance from VIII as the hour hand is from XII?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 40968,
        "difficulty": null
      },
      {
        "id": "afc5dc9821517521f933c9acbb5e4926e46d00cd98fead9f4d70b60dd252f68b",
        "domain": "numina",
        "question": "6. Solve the system of equations for positive real numbers:\n\n$$\n\\frac{1}{x y}=\\frac{x}{z}+1, \\quad \\frac{1}{y z}=\\frac{y}{x}+1, \\quad \\frac{1}{z x}=\\frac{z}{y}+1\n$$",
        "answer": "\\frac{1}{\\sqrt{2}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 180369,
        "difficulty": null
      },
      {
        "id": "32caa9417d92068e68259a31ac906a968db1356b725d54380ad5524807908860",
        "domain": "numina",
        "question": "4. On his birthday, Piglet baked a large cake weighing 10 kg and invited 100 guests. Among them was Winnie-the-Pooh, who is fond of sweets. The birthday boy announced the rule for dividing the cake: the first guest cuts a piece of the cake that is $1 \\%$, the second guest cuts a piece of the cake that is $2 \\%$ of the remaining part, the third guest cuts a piece of the cake that is $3 \\%$ of the remaining part, and so on. What place in line should Winnie-the-Pooh take to get the largest piece of cake?",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14677,
        "difficulty": null
      },
      {
        "id": "4a064e5ba22b8963392d88d91ec7d0e72ed2a61de143bc65a2bfb68a3839d449",
        "domain": "numina",
        "question": "## Exercise 1. Calculate\n\n$$\n\\sqrt{32 \\sqrt{16 \\sqrt{8 \\sqrt{4}}}}\n$$\n\nOnly a numerical answer is expected here.",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 78268,
        "difficulty": null
      },
      {
        "id": "97bb9c88ee2ad03384e5f31999515fa9700170409691c32c58aa5f5fa2ec7914",
        "domain": "numina",
        "question": "8. In a $5 \\times 5$ matrix, each element is either 0 or 1, and it satisfies: the sum of the elements in each of the five rows is equal, but the sum of the elements in the five columns are all different. The number of such matrices is $\\qquad$",
        "answer": "26400",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 138772,
        "difficulty": null
      },
      {
        "id": "b3bcc67e7ca609ecf69c30e56d8a4d5844a81b5ea6f413082073b8f57937753a",
        "domain": "numina",
        "question": "## Task 36/79\n\nSolve the equation\n\n$$\n\\binom{4 k}{k} \\cdot k^{2}=\\binom{4 k}{k+1} \\cdot(k+1)\n$$\n\nwith $k \\in \\mathbb{N}, k>0$, and calculate the value of both terms for the found solution(s).",
        "answer": "1980",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 182594,
        "difficulty": null
      },
      {
        "id": "7056bbca641e92f86b4673b8256d06ad8b9301f843bf5aa83af573b6134c7396",
        "domain": "numina",
        "question": "A parttal párhuzamosan, attól 200 méterre halad egy vitorlás a Balatonon. Valaki folyamatosan egy irányban úszva szeretné elérni a közeledő hajót. A parthoz képest milyen szögben kell elindulnia, ha a vitorlás sebessége $15 \\mathrm{~km} / \\mathrm{h}$, az úszó sebessége $2 \\mathrm{~km} / \\mathrm{h}$, és induláskor a parton mérve $2 \\mathrm{~km}$ távolságban van a hajótól？\n\nA boat is sailing parallel to the shore, 200 meters away from it, on Lake Balaton. Someone wants to swim in a straight line to reach the approaching boat. At what angle relative to the shore should they start swimming if the boat's speed is $15 \\mathrm{~km} / \\mathrm{h}$, the swimmer's speed is $2 \\mathrm{~km} / \\mathrm{h}$, and at the start, they are $2 \\mathrm{~km}$ away from the boat, measured along the shore?",
        "answer": "42.54",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 169705,
        "difficulty": null
      },
      {
        "id": "6b4acd3ad93e0ef7496e7660e7b2a826a76c8ff81e9401166f623f76da81fe65",
        "domain": "numina",
        "question": "B2. Let\n\n$$\nf(x)=\\ln x \\quad \\text { and } \\quad g(x)=\\frac{1}{3} x^{3}-x^{2}-6 x+4\n$$\n\nbe two functions. Let $p$ be the tangent to the graph of $f$ at the point $S(a, b)$, and $q$ be the tangent to the graph of $g$ at the point $T(a, c)$. The tangent $p$ is perpendicular to the tangent $q$. Calculate the value of $a$.",
        "answer": "3",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 164175,
        "difficulty": null
      },
      {
        "id": "1fc8990841686882873b3f31bcf4058bf12a23ad80367635d481a967fed1426d",
        "domain": "numina",
        "question": "1. Simplify:\n$$\n\\sum_{k=1}^{2016}(k \\sqrt{k+1}+(k+1) \\sqrt{k})^{-1}=\n$$",
        "answer": "1-\\frac{1}{\\sqrt{2017}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94208,
        "difficulty": null
      },
      {
        "id": "10189321cd816f45882412a7f4e3670cd424cc1e69637c351db017dcf3d1976c",
        "domain": "numina",
        "question": "Problem 7.1.1. Let $A, B, C, D, E, F, G, H$ - be different digits from 0 to 7 - satisfy the equation\n\n$$\n\\overline{A B C}+\\overline{D E}=\\overline{F G H}\n$$\n\nFind $\\overline{D E}$, if $\\overline{A B C}=146$.\n\n(The notation $\\overline{A B C}$ represents a three-digit number consisting of digits $A, B, C$, similarly constructed are the numbers $\\overline{F G H}$ and $\\overline{D E$).",
        "answer": "57",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16015,
        "difficulty": null
      },
      {
        "id": "d248f57f55383cb391886b1db9fcc1c9ffbb0d55a445956bbfff33ed50529d67",
        "domain": "numina",
        "question": "Point $O$ divides the segment $A B$ into segments $O A=6$ and $O B=4$. A circle is drawn with center at point $O$, and tangents are drawn from $A$ and $B$ to the circle, intersecting at point $M$, with the points of tangency lying on the same side of the line $A B$. Find the radius of the circle if $O M=12$.",
        "answer": "\\frac{6\\sqrt{21}}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62800,
        "difficulty": null
      },
      {
        "id": "febb9a1336fedfe0e6a5a775a64169dcaf61eb38a12edb85c7261aa00ee39a09",
        "domain": "numina",
        "question": "## Task 3 - 100923\n\nGiven are two real numbers $m \\neq 0$ and $n$. Furthermore, let $f$ be the function defined by $f(x)=m x+n$ for all real numbers.\n\na) Determine all numbers $x_{0}$ for which $2 \\cdot f\\left(x_{0}\\right)=f\\left(x_{0}+2\\right)$ holds (i.e., for which the function value at $x_{0}+2$ is twice as large as the one at $x_{0}$) when $m=1$ and $n=0$!\n\nb) Determine all numbers $x_{0}$ for which $2 \\cdot f\\left(x_{0}\\right)=f\\left(x_{0}+2\\right)$ holds for any given real numbers $m \\neq 0$ and $n$!",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184486,
        "difficulty": null
      },
      {
        "id": "12146a6e21d2ff6021d7072a5e66632bf27cdee7bf802e3f5f1df8f40bf46a06",
        "domain": "numina",
        "question": "## Task B-3.1.\n\nCalculate: $\\frac{1}{\\left(337^{\\frac{1}{2}}+6^{\\frac{1}{2}}\\right)^{-2}}-\\left(\\frac{\\sqrt{337}-\\sqrt{6}}{337^{\\frac{3}{2}}-6^{\\frac{3}{2}}}\\right)^{-1}$.",
        "answer": "\\sqrt{2022}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88234,
        "difficulty": null
      },
      {
        "id": "61ef84176adb914970d681f233bed5c53deb7d32595cdf9e383e010828d561d4",
        "domain": "numina",
        "question": "A circle with center $O$ is inscribed with a triangle $ABC (A>90^{\\circ})$. The extension of the angle bisector $AF$ of angle $A$ of this triangle intersects the circle at point $L$, and the radius $AO$ intersects the side $BC$ at point $E$. Let $AH$ be the altitude of triangle $ABC$. Find the ratio of the area of triangle $OAL$ to the area of quadrilateral $OEF L$, given that $AL=4\\sqrt{2}$, $AH=\\sqrt{2\\sqrt{3}}$, and $\\angle AEH=60^{\\circ}$.",
        "answer": "\\frac{4}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57835,
        "difficulty": null
      },
      {
        "id": "e882fe9c4945a8f0db2340e36d72cdcb5228d04e0f480cdb4735baca3261d3e2",
        "domain": "numina",
        "question": "13.361. A red pencil costs 27 kopecks, a blue one - 23 kopecks. No more than 9 rubles 40 kopecks can be spent on purchasing pencils. It is necessary to purchase the maximum possible total number of red and blue pencils. At the same time, the number of red pencils should be as few as possible, but the number of blue pencils should not differ from the number of red pencils by more than 10. How many red and how many blue pencils should be purchased under the given conditions?",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49445,
        "difficulty": null
      },
      {
        "id": "4af7645a6537d3e7911197ae720e4ac8f719906df36afbc75df4d3b97754960e",
        "domain": "numina",
        "question": "40. In an equilateral triangle $ABC$ with a side length of 6 (Fig. 42), find the scalar product of the vectors: a) $\\overrightarrow{A B}$ and $\\overrightarrow{A C} ;$ b) $\\overrightarrow{A B}$ and $\\overrightarrow{B C}$.",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 31872,
        "difficulty": null
      },
      {
        "id": "d17415bea55cfb67f3eca8e829d4ecfd37ff2a61325321661138da27f1700c62",
        "domain": "numina",
        "question": "7. The number $n+2015$ is divisible by 2016, and the number $n+2016$ is divisible by 2015. Find the smallest natural $n$ for which this is possible.\n\nANS: 4058209.",
        "answer": "4058209",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 5604,
        "difficulty": null
      },
      {
        "id": "19e53d1df9d2036fe679eeee6c4e56694de633e450a5553d5f6ab9e3d2b899ee",
        "domain": "numina",
        "question": "4. (4 points) In a parallelogram with sides 4 and 7, the bisectors of the four interior angles are drawn. Find the ratio of the area of the quadrilateral formed by the intersection of the bisectors to the area of the parallelogram.",
        "answer": "\\frac{9}{56}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8651,
        "difficulty": null
      },
      {
        "id": "8ebc6cd14dd7ec0f40fd6c29e11333d5bbf58ccf43998c939c31370066ec74e3",
        "domain": "numina",
        "question": "$3+$\n\n$p$ and $8 p^{2}+1$ are prime numbers. Find $p$.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 67985,
        "difficulty": null
      },
      {
        "id": "e24f3061eadcfbfb72a226de33819887b480cc1e2fefe412c63782447e46dc61",
        "domain": "numina",
        "question": "7. In $\\triangle A B C$, it is known that $a+c=2 b$, then $\\operatorname{tg} \\frac{A}{2} \\operatorname{tg} \\frac{C}{2}=$ $\\qquad$ .",
        "answer": "\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 128724,
        "difficulty": null
      },
      {
        "id": "a5f6d9236e74a68bf53fe0abd67d0f0068d44e84d1aebe147fa6fbf1eb839560",
        "domain": "numina",
        "question": "1. A sequence of real numbers $a_{0}, a_{1}, a_{2}, \\ldots$ satisfies the relation\n\n$$\na_{m+n}+a_{m-n}-m+n-1=\\frac{a_{2 m}+a_{2 n}}{2}\n$$\n\nfor all $m, n \\in \\mathbb{N}, m \\geqslant n$. If $a_{1}=3$, find $a_{2004}$.\n\n(Kunap)",
        "answer": "2004^2+2004+1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 161379,
        "difficulty": null
      },
      {
        "id": "082867a4fa46e82796f7b2f1acfb279c07a6b2e0045a674148384ee33e19373d",
        "domain": "numina",
        "question": "Problem 6.3. Three merchants: Foma, Yerema, and Julius met in Novgorod. If Foma gives Yerema 70 gold coins, then Yerema and Julius will have the same amount of money. If Foma gives Yerema 40 gold coins, then Foma and Julius will have the same amount of money. How many gold coins should Foma give Yerema so that they both have the same amount of money?",
        "answer": "55",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12875,
        "difficulty": null
      },
      {
        "id": "51e13d7aab55a6c6843ba11a51e19614080d0ca1c64199d4a45280ea81a87dc1",
        "domain": "numina",
        "question": "8. Let $\\{a, b, c, d\\}$ be a subset of $\\{1,2, \\cdots, 17\\}$. If 17 divides $(a-b+c-d)$, then $\\{a, b, c, d\\}$ is called a \"good subset\". Then, the number of good subsets is $\\qquad$",
        "answer": "476",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123752,
        "difficulty": null
      },
      {
        "id": "957c58c5ecc84e82a5703dc7d15618df290fa4aadcbdf3b96457bebaab277b6c",
        "domain": "numina",
        "question": "Example 8 Find the maximum value of $n$ such that there exists an arithmetic sequence $a_{1}, a_{2}, \\cdots, a_{n}(n \\geqslant 3)$ satisfying\n$$\n\\sum_{i=1}^{n}\\left|a_{i}\\right|=\\sum_{i=1}^{n}\\left|a_{i}+1\\right|=\\sum_{i=1}^{n}\\left|a_{i}-2\\right|=507 .\n$$\n(2005, China Southeast Mathematical Olympiad)",
        "answer": "26",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113478,
        "difficulty": null
      },
      {
        "id": "fa0be8dd6a911755d870ce39744da36ec6be267db3311dc589e635eb4cd9fbca",
        "domain": "numina",
        "question": "In triangle $ABC$, with the usual notations, $a > b$, and the segments $f_{1}$ and $f_{2}$, which are the distances from the internal and external angle bisectors of angle $C$ to the line $AB$, respectively, satisfy\n\n$$\n\\frac{f_{2}}{f_{1}}=\\frac{a+b}{a-b} \\sqrt{3}\n$$\n\nDetermine the angle at $C$.",
        "answer": "120",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 168563,
        "difficulty": null
      },
      {
        "id": "524abd4a9140cdacb28791c25d8a33a8132333ea82c7e707e9f783db867ae72b",
        "domain": "numina",
        "question": "G8.2 If the area of the equilateral triangle $P Q R$ is $6+b \\sqrt{3}$, find the value of $b$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152775,
        "difficulty": null
      },
      {
        "id": "fe175f0a8ad47a940e00d113ae45b0b5198c6a83e3d480adfe066e0f66ff2502",
        "domain": "numina",
        "question": "2.298. $\\frac{\\sqrt{21+8 \\sqrt{5}}}{4+\\sqrt{5}} \\cdot \\sqrt{9-4 \\sqrt{5}}=\\sqrt{5}-2$.",
        "answer": "\\sqrt{5}-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47965,
        "difficulty": null
      },
      {
        "id": "4927aaaa80a7b3304270d5469f2348ccedeedfa45b8f96daf54b2e9cf29100d7",
        "domain": "numina",
        "question": "An Eilitnip number is a three-digit positive integer with the properties that, for some integer $k$ with $0 \\leq k \\leq 7$ :\n\n- its digits are $k, k+1$ and $k+2$ in some order, and\n- it is divisible by $k+3$.\n\nDetermine the number of Eilitnip numbers.",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194052,
        "difficulty": null
      },
      {
        "id": "b23c580051e3953f7166fd766bff966515680282f48ade1093ed3b7b01e8d2f1",
        "domain": "numina",
        "question": "(15) Real numbers $x, y, z$ satisfy $x^{2}+y^{2}+z^{2}=1$, then the maximum value of $\\sqrt{2} x y+y z$ is\n$\\qquad$ .",
        "answer": "\\frac{\\sqrt{3}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105963,
        "difficulty": null
      },
      {
        "id": "1bc21e4b0d2762ddd2a51db4e10e265a6b5f482036a1f2944c3a15db64aeb7a9",
        "domain": "numina",
        "question": "Question 25, Given that the graph of the function $y=f(x)$ is centrally symmetric about the point $(1,1)$ and axially symmetric about the line $x+y=0$. If $x \\in(0,1)$, $f(x)=\\log _{2}(x+1)$, then the value of $f\\left(\\log _{2} 10\\right)$ is $\\qquad$ -",
        "answer": "\\frac{17}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133629,
        "difficulty": null
      },
      {
        "id": "c7bf2daae925cdc732a6b8f4619520d9090be939be8308f9effc8f85ad7b237c",
        "domain": "numina",
        "question": "(5) Find the largest real number $\\lambda$ such that for a real-coefficient polynomial $f(x)=x^{3}+$ $a x^{2}+b x+c$ with all roots being non-negative real numbers, if $x \\geqslant 0$, then\n$$\nf(x) \\geqslant \\lambda(x-a)^{3}\n$$\n\nWhen does the equality hold in the above inequality?",
        "answer": "-\\frac{1}{27}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109842,
        "difficulty": null
      },
      {
        "id": "a1e27ac4c1e40bef8b457defefbb579715bb858b51ae0f459da51f514cb32215",
        "domain": "numina",
        "question": "The circle constructed on the larger lateral side $AB$ of the right trapezoid $ABCD$ as its diameter intersects the base $AD$ at its midpoint. It is known that $AB=10, CD=6$. Find the midline of the trapezoid.",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67594,
        "difficulty": null
      },
      {
        "id": "69d0a00abc8985639339d7a257b04209e8242f5e761cd834222328dfb972badc",
        "domain": "numina",
        "question": "120. Find $\\lim _{x \\rightarrow 3}\\left(x^{2}-7 x+4\\right)$.",
        "answer": "-8",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31942,
        "difficulty": null
      },
      {
        "id": "f2df4538de377ccda4013009a60246e85fedeca6ef3278fe9598afa7c0506b0e",
        "domain": "numina",
        "question": "2. Let ABCD be a parallelogram and E, F the midpoints of segments [BC], and respectively [CD]. If $\\mathrm{AE} \\cap \\mathrm{BF}=\\{\\mathrm{G}\\}$, and $\\mathrm{H} \\in(\\mathrm{AG}), \\mathrm{with}[\\mathrm{AH}] \\equiv[\\mathrm{HG}]$, find $\\frac{H G}{H E}$.\n\n(Sorin Furtună, Stelică Pană, Olympiads and Mathematics Competitions V-VIII 2014, Bîrchi Publishing)",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 179276,
        "difficulty": null
      },
      {
        "id": "5f6927df65418faa56b6a791b1e118d388f0700619efdd84131d5c72576f703c",
        "domain": "numina",
        "question": "Task 2. The houses on a street are numbered from 1 to 100. How many times does the digit $7$ appear in the house numbers?",
        "answer": "20",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188964,
        "difficulty": null
      },
      {
        "id": "0edbecc4850aab66198bfa7b3944027d0ab092ca13ce4f5d20ac44339dd03cd6",
        "domain": "numina",
        "question": "3. In the number $2 * 0 * 1 * 6 * 0 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,1,2,3,4,5,6,7,8$ (digits can repeat) so that the resulting 10-digit number is divisible by 45. In how many ways can this be done?",
        "answer": "1458",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 2909,
        "difficulty": null
      },
      {
        "id": "f746d444aa71c88f31939624dad7f4c74cf258b398ea11d13af3a7bcefe2c0dd",
        "domain": "numina",
        "question": "Problem 4.3. Zhenya drew a square with a side of 3 cm, and then erased one of these sides. A figure in the shape of the letter \"P\" was obtained. The teacher asked Zhenya to place dots along this letter \"P\", starting from the edge, so that the next dot was 1 cm away from the previous one, as shown in the picture, and then count how many dots he got. He got 10 dots.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5b323dccc142e96a9907g-03.jpg?height=331&width=329&top_left_y=196&top_left_x=562)\n\nThen the teacher decided to complicate the task and asked to count the number of dots, but for the letter \"P\" obtained in the same way from a square with a side of 10 cm. How many dots will Zhenya have this time?",
        "answer": "31",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17914,
        "difficulty": null
      },
      {
        "id": "be39f4845312b025232dfb9d589e2d68657e85064d06700695934e5da596d452",
        "domain": "numina",
        "question": "Example 4 Find the number of solutions to $\\sin x=\\lg x$.\nAnalysis This is a transcendental equation, which cannot be solved using elementary methods (thus making it impossible to determine the number of solutions directly).\nLet $y_{1}=\\sin x=f_{1}(x), y_{2}=\\lg x=f_{2}(x)$, then the number of solutions to the original equation is converted to the number of intersection points of the graphs of the two functions $f_{1}(x)$ and $f_{2}(x)$. The discussion of the existence and number of intersection points within a certain range relies on the algebraic analysis of the properties of the functions.",
        "answer": "31",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116144,
        "difficulty": null
      },
      {
        "id": "4a217f801ff3e19b0ec1ec7ba3f5a9afcea53472d2d632584a8d69f6f1015972",
        "domain": "numina",
        "question": "A fair die is rolled six times. Find the mathematical expectation of the number of different faces that appear.\n\n#",
        "answer": "\\frac{6^{6}-5^{6}}{6^{5}}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55867,
        "difficulty": null
      },
      {
        "id": "bdea83762014746ea693809ed7217d3d5fecca1c6b54eff5de3b408d331ab6c5",
        "domain": "numina",
        "question": "3. Apples were divided into two unequal piles. When half of the apples from the first pile were moved to the second pile, and then half of the apples from the second pile were moved back to the first pile, there were 18 apples in the first pile and 8 in the second. How many apples were in each pile originally?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42729,
        "difficulty": null
      },
      {
        "id": "aad0c213a92945b8028aa816e1023e7f1c8c93ba2d6d322ed2da91fbfe9144f7",
        "domain": "numina",
        "question": "Consider the points $P(0,0), Q(4,0)$ and $R(1,2)$. The line with equation $y=m x+b$ is parallel to $P R$ and passes through the midpoint of $Q R$. What is the value of $b$ ?",
        "answer": "-4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194014,
        "difficulty": null
      },
      {
        "id": "14854c0698851100c5410d42db11415e4f0139d735aeeb3409b1ed97dc5116b1",
        "domain": "numina",
        "question": "10. The last digit of a four-digit number is 2. If the last 2 is moved to the first position, the new number is 180 greater than the original number. The original number is $\\qquad$ -",
        "answer": "2022",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 139360,
        "difficulty": null
      },
      {
        "id": "91dfbea2e63cff94318e1cef45892b66a0cde9186790589c9754641de213ba3a",
        "domain": "numina",
        "question": "8. If the internal radius $R=2+\\sqrt{6} \\mathrm{~cm}$ of a hollow sphere contains 4 solid spheres of radius $r$, then the maximum value that $r$ can take is\n\n保留了原文的换行和格式。",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126129,
        "difficulty": null
      },
      {
        "id": "5a0cb327c44064b161fc5b330497e524d5d7fd016ff792aa61122253d696db35",
        "domain": "numina",
        "question": "Problem 9. The function $G$ is defined on the set of triples of integers and takes real values. It is known that for any four integers $a, b, c$ and $n$, the equalities $G(n a, n b, n c)=n \\cdot G(a, b, c)$, $G(a+n, b+n, c+n)=G(a, b, c)+n$, $G(a, b, c)=G(c, b, a)$ hold. Find $G(89,90,91)$.",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9413,
        "difficulty": null
      },
      {
        "id": "41edf7ffb7140a77da55516d851be5ecbde08a88adc7daaa18aff22eac2a9cd4",
        "domain": "numina",
        "question": "8. [6] $A B C D$ is a convex quadrilateral such that $A B<\\underline{A D}$. The diagonal $\\overline{A C}$ bisects $\\angle B A D$, and $m \\angle A B D=130^{\\circ}$. Let $E$ be a point on the interior of $\\overline{A D}$, and $m \\angle B A D=40^{\\circ}$. Given that $B C=$ $C D=D E$, determine $m \\angle A C E$ in degrees.",
        "answer": "55",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150872,
        "difficulty": null
      },
      {
        "id": "1e773c32ca2aa3f32a7377f80778da5598cfb030d8f82439a194522906630d0f",
        "domain": "numina",
        "question": "[ Identical Transformations ]\n\nRepresent the numerical expression $2 \\cdot 2009^{2}+2 \\cdot 2010^{2}$ as the sum of squares of two natural numbers.\n\n#",
        "answer": "4019^{2}+1^{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69464,
        "difficulty": null
      },
      {
        "id": "8d1387fbfe2ee5a36222efc226184a09688ad4bd5becc7854a466e656ec73da6",
        "domain": "numina",
        "question": "3. Find the last digit of the integer part of the number $(\\sqrt{37}+\\sqrt{35})^{2016}$.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 20428,
        "difficulty": null
      },
      {
        "id": "4dc119b2f7d9b208a7ab321f96bf6a276c34255cb59ea875685197d3378e2a55",
        "domain": "numina",
        "question": "4、A row of 20 seats, some of which are already occupied. If one more person sits down, there will be two adjacent seats occupied. The minimum number of people already seated is\n$\\qquad$ people",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 124032,
        "difficulty": null
      },
      {
        "id": "40aa1d7d72c73e3266824ff47ba593a9fabc20708ca8633dfca84c44d2d7e748",
        "domain": "numina",
        "question": "11. Let $S(k)$ denote the sum of all the digits in the decimal representation of a positive integer $k$. Let $n$ be the smallest positive integer satisfying the condition $S(n)+S(n+1)=$ param1. As the answer to the problem, write down a five-digit number such that its first two digits coincide with the first two digits of $n$ and its last three digits coincide with the last three digits of $n$. For example, if $\\mathrm{n}=1234567890$, then the answer must be 12890.\n\n| param1 | Answer |\n| :---: | :---: |\n| 2016 | 59989 |\n| 664 | 49989 |\n| 1580 | 39989 |\n| 4000 | 79989 |\n\n## Condition\n\nLet $S(k)$ denote the sum of all the digits of the number $k$. Let $n$ be the smallest natural number such that $S(n)+S(n+1)=2016$. As the answer to the problem, write down a five-digit number such that its first two digits coincide with the first two digits of $n$, and its last three digits coincide with the last three digits of $n$. For example, if $\\mathrm{n}=1234567890$, then the answer should be 12890.\n\n## Answer.\n\n59989\n\n## Solution\n\nIf the number $n$ does not end in 9, then $S(n+1)=S(n)+1$, and the sum $S(n)+S(n+1)$ is odd, which means it cannot equal 2016. Therefore, the number $n$ must end in 9. If it ends in one 9, then $S(n+1)=S(n)-8$, and thus $S(n)=1012$. The smallest number with such a sum of digits is 599...989 (the total number of 9s is 111). If the number of 9s at the end of $n$ is not less than two, then by similar reasoning, the sum $S(n)$ will be larger, and the number $n$ will also be larger. Therefore, the answer is 59989.\n\n12. Let $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ such that $A M=A B$ and $C K=C B$. It is given that the radius of the circumcircle of triangle $K B M$ is param1, and the radius of the incircle of triangle $A B C$ is param2, and this incircle touches side $B C$ at point $T$. Find the square of the length of segment $B T$.\n\nLet $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ so that $A M=$ $A B$ and $C K=C B$. It is given that the radius of the circumcircle of the triangle $K B M$ equals param1, and the radius of the incircle of the triangle $A B C$ equals param2. Find the length of $B T$ squared if $T$ is the point where the incircle of $A B C$ touches its side $B C$.\n\n| param1 | param2 | Answer |\n| :---: | :---: | :---: |\n| 7 | 5 | 24 |\n| 8 | 5 | 39 |\n| 9 | 7 | 32 |\n| 11 | 8 | 57 |\n| 13 | 9 | 88 |\n\n## Condition\n\nLet $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ such that $A M=A B$ and $C K=C B$. It is given that the radius of the circumcircle of triangle $K B M$ is 7, and the radius of the incircle of triangle $A B C$ is 5, and this incircle touches side $B C$ at point $T$. Find the square of the length of segment $B T$.\n\n## Answer.\n\n24",
        "answer": "24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22977,
        "difficulty": null
      },
      {
        "id": "a8af16658b4cfefc18754a267f4ce87ad5d600897d8d1c9b872d0bee8fcc8a8d",
        "domain": "numina",
        "question": "8. Let $F_{1}$ and $F_{2}$ be the two foci of the ellipse $\\frac{x^{2}}{9}+\\frac{y^{2}}{4}=1$, and $P$ be a point on the ellipse such that $\\left|P F_{1}\\right|:\\left|P F_{2}\\right|$ $=2: 1$. Then the area of $\\triangle P F_{1} F_{2}$ is $\\qquad$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126150,
        "difficulty": null
      },
      {
        "id": "cb3a8c2f0a245a87f0a1a4dbae425ae2422623b5b1c91bc668abbd1f3ef9111d",
        "domain": "numina",
        "question": "## Task 4 - 270614\n\nKerstin cuts a rectangular piece of paper so that the cut runs from the midpoint of one rectangular side to the midpoint of the opposite side. This results in two smaller rectangles; Kerstin places them exactly on top of each other so that only a smaller rectangle is visible, consisting of two layers.\n\nKerstin cuts this two-layer rectangle in the same way and again places the resulting rectangular pieces exactly on top of each other. This process continues: overlapping rectangles are cut, and the resulting pieces are placed on top of each other.\n\n(Of course, this cannot be done indefinitely; the stack of paper to be cut becomes thicker each time.)\n\n(1) State the number of paper pieces that have been created after the\n\na) second, b) third, c) fourth cutting!\n\n(2) Assuming that cutting could be done indefinitely, why would it still be impossible for exactly 160 paper pieces to ever result from a single cut?",
        "answer": "160",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183371,
        "difficulty": null
      },
      {
        "id": "b1fe376d4969b17232b319f2eedaae0c12d69b5969a563b297bd7ccce2dd98b5",
        "domain": "numina",
        "question": "In the quadratic equation $x^{2}+z_{1} x+z_{2}+m=0$, $z_{1}, z_{2}, m$ are all complex numbers, and $z_{1}^{2}-4 z_{2}=16+20 \\mathrm{i}$. Suppose the two roots $\\alpha, \\beta$ of this equation satisfy $|\\alpha-\\beta|=$ $2 \\sqrt{7}$, find the maximum and minimum values of $|m|$.",
        "answer": "\\sqrt{41}+7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142655,
        "difficulty": null
      },
      {
        "id": "cb34c8ca0737f20f93d14d4fb826dbf9a694d38444cf633296d42dfdccfd669d",
        "domain": "numina",
        "question": "23. [12] If $x$ and $y$ are real numbers such that $\\frac{(x-4)^{2}}{4}+\\frac{y^{2}}{9}=1$, find the largest possible value of $\\frac{x^{2}}{4}+\\frac{y^{2}}{9}$.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151054,
        "difficulty": null
      },
      {
        "id": "0726136065461352b9aa85b8eb93dc5ec61adc7ca42a3d73e6725e07d444f7db",
        "domain": "numina",
        "question": "2. (i) Expand and simplify $(x+1)\\left(x^{2}-x+1\\right)$.\n(ii) Find all powers of 2 that are one more than a cube.\n(A power of 2 is a number that can be written in the form $2^{n}$, where $n$ is an integer greater than or equal to 0 .)",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156064,
        "difficulty": null
      },
      {
        "id": "40814dc7d4f650f1b226e88ef648b2d0dadb92cb3fff88c34f12945bb27915ac",
        "domain": "numina",
        "question": "Task A-3.4. (8 points)\n\nIf for the lengths $a, b, c$ of the sides of a triangle it holds that $(a+b+c)(a+b-c)=3 a b$, determine the angle opposite side $c$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89388,
        "difficulty": null
      },
      {
        "id": "d409333b428c6e8475f190425afd7809032afb843e244f3d681182348e0bf998",
        "domain": "numina",
        "question": "1. As shown in the figure, in $\\triangle A B C$, $\\angle A=60^{\\circ}, A B>A C$, point $O$ is the circumcenter. The two altitudes $B E$ and $C F$ intersect at point $H$. Points $M$ and $N$ lie on segments $B H$ and $H F$ respectively, and satisfy $B M=C N$. Find the value of $\\frac{M H+N H}{O H}$.",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 120605,
        "difficulty": null
      },
      {
        "id": "089b2d74b2c4906e68877866c014a43edaadbc343ac3a756a7a6ab0789d3e576",
        "domain": "numina",
        "question": "4. (6) In each cell of a $100 \\times 100$ table, a number from the interval $[-1,1]$ is written. The sum of all the numbers is 0, the sum of all the numbers in each row in absolute value is at least $c$, and the sum of all the numbers in each column in absolute value is at least $c$. Find the largest possible value of $c$.",
        "answer": "50",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162016,
        "difficulty": null
      },
      {
        "id": "4e8317a13b75ccde8f5bb0322a2110400853f3c1405483f78aaa00581ee9e1e6",
        "domain": "numina",
        "question": "60. In a parallelogram, the distances from a point inside to the four sides are $1,2,3,4$, then the minimum area of such a parallelogram is . $\\qquad$",
        "answer": "21",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 136218,
        "difficulty": null
      },
      {
        "id": "b1adf4ee759725dbf88f297c5c731f80202fa742c70e6e4bee7b44dae5445dcd",
        "domain": "numina",
        "question": "8. (10 points) In a plane, there are 5 points, among which no 3 points are collinear. Connecting line segments with these points as endpoints, apart from these 5 points, these line segments have at least $\\qquad$ intersection points.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 116583,
        "difficulty": null
      },
      {
        "id": "95934a9c144ba4f562bfbb3db7e48399676f6c1cd458fceb6eb737a23bae7c31",
        "domain": "numina",
        "question": "4. Given complex numbers $z, z_{1}, z_{2}\\left(z_{1} \\neq z_{2}\\right)$ satisfy:\n$$\nz_{1}^{2}=z_{2}^{2}=-2-2 \\sqrt{3} \\mathrm{i},\\left|z-z_{1}\\right|=\\left|z-z_{2}\\right|=4 \\text {. }\n$$\n\nThen $|z|=$ $\\qquad$",
        "answer": "2\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146620,
        "difficulty": null
      },
      {
        "id": "6a58150dbe7d4f5c289b74f15be506459d2dd2ee14c0730915ab816ede23d870",
        "domain": "numina",
        "question": "8. (3 points) Fill in the circles in the figure with $10, 15, 20, 30, 40$, and $60$ so that the product of the 3 numbers at the vertices of the three small triangles $A, B, C$ are all equal. What is the maximum equal product? $\\qquad$ .",
        "answer": "18000",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 121382,
        "difficulty": null
      },
      {
        "id": "59111ed3e197bd49881dfd146fd0178a697d7f0fda1010e4d0397df726dc1491",
        "domain": "numina",
        "question": "33(1109). Find the members of the proportion $x_{1}: x_{2}=x_{3}: x_{4}$, where the first term is 6 more than the second, and the third is 5 more than the fourth. The sum of the squares of all terms is 793.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35981,
        "difficulty": null
      },
      {
        "id": "fd41f78c3a857ec8a92dfeb1ef68acd14f03f60257b446eeffcbe13231700bff",
        "domain": "numina",
        "question": "7. (3 points) A rectangular wooden block with a length of 25 cm, a width of 10 cm, and a height of 4 cm is sawed into several equal-sized cubes, which are then assembled into a large cube. The surface area of this large cube is $\\qquad$ square centimeters.",
        "answer": "600",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 128630,
        "difficulty": null
      },
      {
        "id": "06f5bf886436d0dc0674c565a941e7d292c2a0cf31f7a37ef373cf6a9f545e53",
        "domain": "numina",
        "question": "Example 1 Solve the equation: $\\lg (3-x)-\\lg (3+x)=\\lg (1-x)-\\lg (2 x+1)$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122432,
        "difficulty": null
      },
      {
        "id": "fc525d8e1641f168d702960de23a8ad0019fba72a1121992938d0c44422dbc21",
        "domain": "numina",
        "question": "Two boxes together contain 65 balls of various sizes. Each ball is white, black, red, or yellow. Each time we take five balls of the same color, at least two are of the same size.\n\n(a) What is the maximum number of types of balls that exist in the boxes? Two balls are considered of different types when they have different colors or sizes.\n\n(b) Show that there are at least three balls, which are in the same box, and which are of the same type.\n\nHint: Observe that for a token to be removed it had to be flipped an odd number of times, and all circles have an even number of neighbors.\nHint: There are at most 4 distinct sizes of balls for each color.\n\nHint: Being a/600 and b/700 the two fractions, check which factors the numerator and the denominator of the sum can have in common.\n\nFacts that Help: A fraction is said to be irreducible if the numerator and the denominator do not have common prime factors.\nHint: Observe that the particular values of \\(x_{1}, x_{2}, \\ldots, x_{n}\\) are not important, but rather the number of these that are equal to 1 and -2.",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73075,
        "difficulty": null
      },
      {
        "id": "a27d35b987f654b0b16f7ff730e87873483281d72ad15d4519bcafee3f201c4c",
        "domain": "numina",
        "question": "Gapochkin A.i.\n\nHow many integers from 1 to 1997 have a sum of digits that is divisible by 5?",
        "answer": "399",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68882,
        "difficulty": null
      },
      {
        "id": "7c595eccf0793afa042cf039f5b5a9b27e22ae6e1ae6f027242652cc3eee6aa2",
        "domain": "numina",
        "question": "Let's determine the distance between the tangents parallel to the line $y=x$ for the parabolas given by the equations $y=x^{2}+1$ and $x=y^{2}+1$.\n\n---\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly. \n\n---\n\nLet's determine the distance between the tangents parallel to the line $y=x$ for the parabolas given by the equations $y=x^{2}+1$ and $x=y^{2}+1$.",
        "answer": "\\frac{3\\sqrt{2}}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171121,
        "difficulty": null
      },
      {
        "id": "983d6e46a1d44ff1f5a37fe64bda31007ab547b1a67ef91ef9e865532c4f45dd",
        "domain": "numina",
        "question": "Adámek was recalculating his collection of rainbow marbles. He found that he could divide them into equal piles in several ways. If he divided them into three piles, there would be $\\mathrm{v}$ more marbles in each pile than if he divided them into four piles.\n\nHow many rainbow marbles did Adámek have?\n\n(E. Semerádová)\n\nHint. Imagine actually regrouping the marbles from four piles into three.",
        "answer": "96",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76580,
        "difficulty": null
      },
      {
        "id": "a05546ca554409a7b753b5a4d755deaa87f0955def16828485987d85d3b4957b",
        "domain": "numina",
        "question": "4. (5 points) Definition: $m \\otimes n=m \\times m-n \\times n$, then $2 \\otimes 4-4 \\otimes 6-6 \\otimes 8-\\cdots-98 \\otimes 100=$",
        "answer": "9972",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140214,
        "difficulty": null
      },
      {
        "id": "af0c870a54ea0fe2c1dfa86536e1e61b475e2fe33ae4d4a8d0197e12341885ac",
        "domain": "numina",
        "question": "12. (3 points) A rectangular photo frame is 40 cm long and 32 cm wide. A photo that is 32 cm long and 28 cm wide is placed inside. The area of the part of the frame not covered by the photo is $\\qquad$ square centimeters.",
        "answer": "384",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 131116,
        "difficulty": null
      },
      {
        "id": "67caccfa542a7deaac22423ee9494541fabb6718996065a9296ba29e0d3c0347",
        "domain": "numina",
        "question": "Solve the following equation:\n\n$$\n\\sqrt{x+\\sqrt{x}}-\\sqrt{x-\\sqrt{x}}=\\frac{3}{2} \\sqrt{\\frac{x}{x+\\sqrt{x}}}\n$$",
        "answer": "\\frac{25}{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167256,
        "difficulty": null
      },
      {
        "id": "4a9fcb4f05160215cf7045bde171423e4522693a012334bbc31cebe1d24f67e9",
        "domain": "numina",
        "question": "[\n\nTwo circles touch each other internally. A line passing through the center of the larger circle intersects it at points $A$ and $D$, and the smaller circle at points $B$ and $C$. Find the ratio of the radii of the circles if $A B: B C: C D=3: 7: 2$.\n\n#",
        "answer": "\\frac{3}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65033,
        "difficulty": null
      },
      {
        "id": "ab3eb8ed1d794d6eea6c254d15219d554843eeb319c31f2d0c9e13e207053baa",
        "domain": "numina",
        "question": "We have a T-shaped figure composed of 22 squares with a side length of $1 \\mathrm{~cm}$. A line $p$ passes through point $V$, dividing the figure into two parts, white and gray. Determine in what ratio the line $p$ divides the segment $A B$, given that the areas of the white and gray regions are equal.\n\n(L. Šimünek)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_89c7e734c425fc1afd89g-1.jpg?height=591&width=593&top_left_y=1943&top_left_x=800)",
        "answer": "\\frac{3}{1}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76225,
        "difficulty": null
      },
      {
        "id": "6f0feba1ff86389d5898eebc299151b7ad723bb1e5434ae414941f8e21029983",
        "domain": "numina",
        "question": "Example 6 Determine an integer $k$ not less than 1, such that the expression $\\sin k x \\cdot \\sin ^{k} x+$ $\\cos k x \\cdot \\cos ^{k} x-\\cos ^{k} 2 x$ is independent of $x$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119985,
        "difficulty": null
      },
      {
        "id": "cd227b214c2a3899c2b564b86a73db3d2b4ad478407855e22ed861f9a23fe49c",
        "domain": "numina",
        "question": "11.4. Given the function $f(x)=\\left(1-x^{3}\\right)^{-1 / 3}$. Find $f(f(f \\ldots f(2018) \\ldots))$ (the function $f$ is applied 2019 times)",
        "answer": "2018",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13532,
        "difficulty": null
      },
      {
        "id": "2ccfd61a7895afc4d972487a26c8d4db32507ac185ad6704fc2465f029458e11",
        "domain": "numina",
        "question": "Paris has two million inhabitants. A human being has, at most, 600000 hairs on their head. What is the largest number of Parisians that we can hope to find who have exactly the same number of hairs on their head?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 79348,
        "difficulty": null
      },
      {
        "id": "8212087991455a1f3e7e348c3011713757dcaa182570a409b49cb589215029bc",
        "domain": "numina",
        "question": "What is the greatest number of acute angles that can occur in a convex polygon?\n\n#",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62612,
        "difficulty": null
      },
      {
        "id": "df39b0f3bae61e13a2f3fbe40a0934fcda82aabdf5ba09444dffca01999b5353",
        "domain": "numina",
        "question": "6. Find the smallest real number $r$ such that a rod of length 1 can be broken into four parts, each of length at most $r$, so that no three of these parts can form a triangle.",
        "answer": "\\frac{3}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75406,
        "difficulty": null
      },
      {
        "id": "fd7903011bdcc1b412863d5c47b36f52d9a3ef8d9a51c879f902b067a1df88e1",
        "domain": "numina",
        "question": "3. Given the sequence $x_{n}$, satisfying $(n+1) x_{n+1}=x_{n}+n$, and $x_{1}=2$, then $x_{2009}=$ $\\qquad$",
        "answer": "\\frac{2009!+1}{2009!}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94381,
        "difficulty": null
      },
      {
        "id": "5d573c9b03e86b9a2bcf23a3cc768af7fa0852a728d8c5c7df1ba2f28d6ca80e",
        "domain": "numina",
        "question": "## 5. Music School\n\nIn a certain music school, each student plays exactly one instrument: piano, guitar, trumpet, or violin. Half of all students play the piano, and half of the half of all students play the guitar. The number of students who play the trumpet is equal to a quarter of the number of students who play the guitar, and 72 students play the violin. How many students are there in the music school?\n\n## Result: $\\quad 384$",
        "answer": "384",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89918,
        "difficulty": null
      },
      {
        "id": "e6d98b951dc24cc3e3705ccc40347a800cb89b1566183f09fe670a52376eaf31",
        "domain": "numina",
        "question": "3. For the tetrahedron $ABCD$, the 6 edge lengths are 7, 13, 18, 27, 36, 41, and it is known that $AB=41$, then $CD=$ $\\qquad$",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144813,
        "difficulty": null
      },
      {
        "id": "a18ca89f3a2f80fe3c1732fa5f8deb51eee675787eeb4b21f31b9dbdd6f83efb",
        "domain": "numina",
        "question": "Problem 3.6. Petya can draw only 4 things: the sun, a ball, a tomato, and a banana. But he does it extremely realistically! Today he drew several things, among which there were exactly 15 yellow, 18 round, and 13 edible. What is the maximum number of balls he could have drawn?\n\nPetya believes that all tomatoes are round and red, all balls are round and can be of any color, and all bananas are yellow and not round.",
        "answer": "18",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13975,
        "difficulty": null
      },
      {
        "id": "7747325998efda9e58c076c2a5b87d8dc253146d95fb1a2cf46ce4510d36a4c2",
        "domain": "numina",
        "question": "12.11. (CSSR, 80). The set $M$ is obtained from the plane by removing three distinct points $A, B$, and $C$. Find the smallest number of convex sets whose union is the set $M$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29015,
        "difficulty": null
      },
      {
        "id": "5c01c40c2e0bfc80c5c054920309d5848d90e600d48d80c1515d7bb4525fa51a",
        "domain": "numina",
        "question": "Example 1. Find $\\lim _{x \\rightarrow 0} \\frac{\\sin x e^{x}-5 x}{4 x^{2}+7 x}$.",
        "answer": "-\\frac{4}{7}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32642,
        "difficulty": null
      },
      {
        "id": "fa225ce4d783f58d55573f0b79cbeb1c0608c62d4666e887984f9c53be434e6a",
        "domain": "numina",
        "question": "[ Law of Cosines $\\quad]$ [ Properties and Criteria of Parallelograms ]\n\nGiven a parallelogram with sides 1 and 2 and an acute angle of $60^{\\circ}$. On two of its opposite sides, isosceles triangles with angles of $120^{\\circ}$ at the vertices are constructed outside the parallelogram. Find the distance between these vertices.",
        "answer": "\\sqrt{\\frac{13}{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60949,
        "difficulty": null
      },
      {
        "id": "ba7ce229b8d0082318ea7d3c5f57f6189e0e6ffd846e76b4334ba6a5daf16c71",
        "domain": "numina",
        "question": "12. (17 points) A rectangular wooden board, if 4 decimeters are cut off from both the longer and shorter sides, the area is reduced by 168 square decimeters. Please ask: What is the original perimeter of the rectangle in decimeters?",
        "answer": "92",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105744,
        "difficulty": null
      },
      {
        "id": "458c372e6b7f03cc5fd667e3efb508f105338b6c8288d0ba4a174f4b02309ef7",
        "domain": "numina",
        "question": "On a circle line, there are 2018 points. Each of these points is labeled with an integer. Each number is greater than the sum of the two numbers that immediately precede it in a clockwise direction.\n\nDetermine the maximum possible number of positive numbers that can be among the 2018 numbers.\n\n(Walther Janous)",
        "answer": "1008",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 190918,
        "difficulty": null
      },
      {
        "id": "26a6413b24a870b6d3d5f3223e037bc31048538aa92b61a15d7d73073f3e509d",
        "domain": "numina",
        "question": "Example 6. Find the variance of the discrete random variable $X$ representing the number of points that fall when tossing a die.",
        "answer": "\\frac{35}{12}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 33023,
        "difficulty": null
      },
      {
        "id": "96416646f51f1999df6d6249d73a9d2fb8c33fe4f206e6a5c2eaa35af37cb3c1",
        "domain": "numina",
        "question": "Bogosiov I.I.\n\n2011 numbers are written on the board. It turns out that the sum of any three of the written numbers is also a written number.\n\nWhat is the smallest number of zeros that can be among these numbers?\n\n#",
        "answer": "2.8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 53667,
        "difficulty": null
      },
      {
        "id": "5e0bf0d641120d5ce80710785383429f4af5af655ad6bbd91f9a94ca1ceff1d2",
        "domain": "numina",
        "question": "# Problem 4. (8 points)\n\nLena receives 50,000 rubles per month and spends 45,000 rubles per month. She gets her salary on the 6th of each month. Lena has: a deposit in the bank $\\mathrm{X}$ with a $1\\%$ monthly interest rate with monthly capitalization and the possibility of topping up, but no withdrawals allowed, with the calculation day being the 6th of each month; a debit card, on which 6% of the minimum balance for the month is credited on the 6th of each month at an annual rate of $8\\%$; a credit card with a payment due date of the 7th of each month and a limit of 100,000 rubles, with a $3\\%$ fee for cash withdrawals or transfers.\n\nLena can pay all her expenses using her bank card. Describe the optimal strategy for Lena and calculate the annual income she can earn from her cooperation with the bank (in rubles).",
        "answer": "4860",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11551,
        "difficulty": null
      },
      {
        "id": "1efba77e01419708e0b5e8b532238006bef632e867e433ae0b3a4b9f2a9c016c",
        "domain": "numina",
        "question": "21. (2004 National College Entrance Examination, Tianjin Paper) From 4 boys and 2 girls, 3 people are randomly selected to participate in a speech contest. Let the random variable $\\xi$ represent the number of girls among the 3 selected people.\n(I) Find the distribution of $\\xi$;\n(II) Find the mathematical expectation of $\\xi$;\n(III) Find the probability that \"the number of girls $\\xi \\leqslant 1$ among the 3 selected people\".",
        "answer": "\\frac{4}{5}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 113591,
        "difficulty": null
      },
      {
        "id": "38afc2fa813d1590e35abbe5223fd0bb11bdca5ddad82532d24ff75a8f52ebd7",
        "domain": "numina",
        "question": "Example 5 Given the set $M=\\left\\{x \\mid x^{2}-2 x+m=0, x \\in \\mathbf{R}\\right\\}$ is non-empty, then the sum of all elements in set $M$ is $\\qquad$ .",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111798,
        "difficulty": null
      },
      {
        "id": "d921d7579ba79273d5c47e256b6b72e48f7cd86157dd9294d4b72a1cd0e2f697",
        "domain": "numina",
        "question": "11. (6 points) It snowed heavily at night, and in the morning, Xiaolong and his father measured the length of a circular path in the garden together, starting from the same point and walking in the same direction. Xiaolong's step length is 54 cm, and his father's step length is 72 cm. After each of them walked one complete circle and returned to the starting point, there were only 60 footprints left in the snow. What is the length of this path? $\\qquad$ meters.",
        "answer": "21.6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 93103,
        "difficulty": null
      },
      {
        "id": "7cf8942d8ab851e42cec20b4fddf5039516827caa470bb869315a15249dff4ed",
        "domain": "numina",
        "question": "21. In a $9 \\times 9$ grid, each small square is colored either black or white, such that the number of black squares adjacent to each white square is greater than the number of white squares, and the number of white squares adjacent to each black square is greater than the number of black squares (squares sharing a common edge are considered adjacent), find the maximum difference in the number of black and white squares in such a coloring method.",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 145388,
        "difficulty": null
      },
      {
        "id": "82e8b7a4c5efdd9ab2bcdecccb98a9626708384bd6947a5e248909ef46bbd726",
        "domain": "numina",
        "question": "1. Given complex numbers $z_{1}, z_{2}$ corresponding to points $A, B$ on the complex plane, and $\\left|z_{1}\\right|=2, z_{1}^{2}-2 z_{1} z_{2}+4 z_{2}^{2}=0, O$ is the origin, then the perimeter of $\\triangle O A B$ is $\\qquad$ .",
        "answer": "3+\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126432,
        "difficulty": null
      },
      {
        "id": "cb7bee6478ac0721fe76965e63b5c43980c6476bdc59c1d8eedad09494eac4b8",
        "domain": "numina",
        "question": "2. Given $x, y \\in\\left[-\\frac{\\pi}{4}, \\frac{\\pi}{4}\\right], a \\in R$, and\n$$\n\\left\\{\\begin{array}{l}\nx^{3}+\\sin x-2 a=0, \\\\\n4 y^{3}+\\sin y \\cdot \\cos y+a=0 .\n\\end{array}\\right.\n$$\n\nThen $\\cos (x+2 y)=$ . $\\qquad$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113823,
        "difficulty": null
      },
      {
        "id": "7b1955a9261996fda54942b3fdb75a61be959bc73aea26f523f83ec894b95917",
        "domain": "numina",
        "question": "6. Try to advance as far as possible in the analytical solution of the problem given below. In case $B$ is necessary, a computer may be used at the final stage.\n\nPoint $A$ is located on a meadow, point $B$ - on a sandy wasteland. The distance between the points is 24 km. The boundary between the wasteland and the meadow is a straight line. The distance from point $A$ to the boundary is 8 km, and the distance from point $B$ to the boundary is 4 km. Find the minimum time it takes for a pedestrian to get from point $A$ to point $B$, if their maximum speed on the wasteland is 3 km/hour, and on the meadow 6 km/hour.",
        "answer": "4.89",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 11246,
        "difficulty": null
      },
      {
        "id": "830d9e5839fc0b5a801f3b25c807f2a4577943996fd896eed2129fc3cd042ebe",
        "domain": "numina",
        "question": "11.071. Find the ratio of the volume of a regular hexagonal pyramid to the volume of a regular triangular pyramid, given that the sides of the bases of these pyramids are equal, and their apothems are twice the length of the sides of the base.",
        "answer": "\\frac{6\\sqrt{1833}}{47}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51360,
        "difficulty": null
      },
      {
        "id": "2b0ba5a24d0b561021168bcfabf2b16eba88887ff550bc17e6b1a83e7a406894",
        "domain": "numina",
        "question": "4. In the sequence $\\left\\{a_{n}\\right\\}$, $a_{k}+a_{k+1}=2 k+1\\left(k \\in \\mathbf{N}^{*}\\right)$, then $a_{1}+a_{100}$ equals",
        "answer": "101",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114878,
        "difficulty": null
      },
      {
        "id": "228eef62ca0be6ecc349c6f6e78caa09f93882787387406e0a6b177ad9f8216a",
        "domain": "numina",
        "question": "9. The Yangtze Farm has a pasture, where grass grows uniformly every day. If 24 cows are grazed on the pasture, they will finish the grass in 6 days; if only 21 cows are grazed, it will take 8 days to finish the grass. How many days will it take to finish the grass if 36 cows are grazed?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107366,
        "difficulty": null
      },
      {
        "id": "213af4e5cba5f9292eef05f33ed82c87b68c13819d9d0d77ca179221ad3e8996",
        "domain": "numina",
        "question": "I1.3 If the equation $x^{2}-c x+b=0$ has two distinct real roots and their difference is 1, determine the greatest possible value of the sum of the roots, $c$.",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153416,
        "difficulty": null
      },
      {
        "id": "6a9d906448c2dea314b032032f20bac829c0e0b48939d57f5b0c1e351b6f29ce",
        "domain": "numina",
        "question": "Suppose that $a, b, c, d$ and $e$ are consecutive positive integers with $a<b<c<d<e$. If $a^{2}+b^{2}+c^{2}=d^{2}+e^{2}$, what is the value of $a$ ?",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 193979,
        "difficulty": null
      },
      {
        "id": "3aaae6106caebbf199fed38bfb641be6b19752171c33cb08ce98ffa9ba23d179",
        "domain": "numina",
        "question": "1. A trip to St. Petersburg is being organized for 30 schoolchildren along with their parents, some of whom will be driving cars. Each car can accommodate 5 people, including the driver. What is the minimum number of parents that need to be invited on the trip?\n\nANSWER: 10",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 4983,
        "difficulty": null
      },
      {
        "id": "d12eab2b5d32feb1681c1d651be53bd8a1a10dd999e9483c095e2812872b0203",
        "domain": "numina",
        "question": "G8.3 If the area of the circle $E F G$ is $\\frac{(c+4 \\sqrt{3}) \\pi}{3}$, find the value of $c$",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152776,
        "difficulty": null
      },
      {
        "id": "74e2d731f03fc1460ac386c084df9d5e6aee68935e441713628b45e9b6317b0e",
        "domain": "numina",
        "question": "14. Harry Potter used a wand to change a fraction, and after the change, he found that the numerator increased by $20 \\%$, and the denominator decreased by $19 \\%$, then the new fraction is increased by $\\qquad$ \\%. (rounded to 1\\%)",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105145,
        "difficulty": null
      },
      {
        "id": "6fb54e2f2ac7eca3771be8aaff444a6aaa72a8802e99fe57e22c214e1e17661f",
        "domain": "numina",
        "question": "$$\n\\text { 4.62 Solve the equation } \\frac{x}{2+\\frac{x}{2+\\frac{x}{2+\\ddots \\frac{x}{2+\\frac{x}{1+\\sqrt{1+x}}}}}}=1\n$$\n(In the above expression, there are 1985 twos.)",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129239,
        "difficulty": null
      },
      {
        "id": "1ff729492879fa242165fbdd1412b398878d1cbe7ffbb97a4df22140f5ca120b",
        "domain": "numina",
        "question": "6. A circle passes through the vertices $K$ and $P$ of triangle $K P M$ and intersects its sides $K M$ and $P M$ at points $F$ and $B$ respectively, such that $K F: F M=3: 1, P B: B M=6: 5$. Find $K P$, if $B F=\\sqrt{15}$.",
        "answer": "2\\sqrt{33}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 2942,
        "difficulty": null
      },
      {
        "id": "76e365a4e0c47266f3af5051662b65eaa2856145e0487957d7bfcc0a226530ae",
        "domain": "numina",
        "question": "Example 7 Given that the sum of several positive integers is 2019. Find the maximum value of their product.",
        "answer": "3^{673}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 98086,
        "difficulty": null
      },
      {
        "id": "2f68e2c376a04b089b9114e7acf18dc847731fe449ad2e4c751b51f160156be8",
        "domain": "numina",
        "question": "3. What are the sizes of the angles of a scalene triangle if the middle angle by size is the arithmetic mean of the remaining two angles and if the length of the longest side of the triangle is twice the length of the shortest side?",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86502,
        "difficulty": null
      },
      {
        "id": "70dd3ba1ae8201e447ee6d4b35dfc7a0806ee3d6ba7605f9e558c0de8f5fbf3d",
        "domain": "numina",
        "question": "For the polynomial\n$$\np(x)=985 x^{2021}+211 x^{2020}-211 \\text {, }\n$$\n\nlet its 2021 complex roots be $x_{1}, x_{2}, \\cdots, x_{2021}$. Then\n$$\n\\sum_{k=1}^{2021} \\frac{1}{x_{k}^{2}+1}=\n$$\n$\\qquad$",
        "answer": "2021",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107479,
        "difficulty": null
      },
      {
        "id": "013c4bd46f76430f545faab88b309463a034285bcfc6c3b2a8d9d3b36abe3d61",
        "domain": "numina",
        "question": "Three, a football invitational tournament involves sixteen cities, each sending two teams, Team A and Team B. According to the competition rules, each pair of teams plays at most one match. Additionally, the two teams from the same city do not play against each other. After a period of matches, it was found that, except for Team A from City A, all other teams have played a different number of matches. How many matches has Team B from City A played? Please prove your conclusion.",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 114618,
        "difficulty": null
      },
      {
        "id": "67c40337a6b0c362182da2138501fb545166a4155d622767e0d95afaebbd2f85",
        "domain": "numina",
        "question": "5. $\\frac{\\sin 7^{\\circ}+\\sin 8^{\\circ} \\cos 15^{\\circ}}{\\cos 7^{\\circ}-\\sin 8^{\\circ} \\sin 15^{\\circ}}=$",
        "answer": "2-\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144379,
        "difficulty": null
      },
      {
        "id": "4052959b2005e382d850cd311863b92414f495e524707b43bb3334c379bed5ca",
        "domain": "numina",
        "question": "5. (10 points) Class 1 of Grade 4 used the class fund to purchase three types of stationery, A, B, and C, with unit prices of 3 yuan, 2 yuan, and 1 yuan, respectively. It is known that the number of B stationery purchased is 2 less than the number of A stationery purchased, and the cost of A stationery does not exceed half of the total cost. If the total cost for the three types of stationery is exactly 66 yuan, then the maximum number of A stationery that can be purchased is $\\qquad$ pieces.",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146190,
        "difficulty": null
      },
      {
        "id": "5d7f215b46a973c801f54372293d3a947186331a4879e35b1e4693d3f91c9294",
        "domain": "numina",
        "question": "16 Given that $\\alpha+\\beta=17$ and $\\alpha \\beta=70$, find the value of $|\\alpha-\\beta|$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160336,
        "difficulty": null
      },
      {
        "id": "2541b9196ff8cc81283c36cddd9445e071507988302b1d1c5e05d8c6420596c9",
        "domain": "numina",
        "question": "![](https://cdn.mathpix.com/cropped/2024_05_06_d3d405875929faed9b27g-21.jpg?height=184&width=2021&top_left_y=917&top_left_x=16)\n\nKoschei the Deathless has kidnapped the three daughters of the tsar. Ivan the Tsarevich set out to rescue them. He comes to Koschei, and Koschei says to him: \"Tomorrow morning, you will see five enchanted girls. Three of them are the tsar's daughters, and the other two are mine. They will be indistinguishable to you, but they will be able to recognize each other. I will approach one of them and ask her about each of the five: 'Is this a tsarevna?' She can answer truthfully or falsely, but she is allowed to call exactly two of them tsarevnas (including herself). Then I will question each of the other girls in the same way, and they must also name exactly two tsarevnas. If you can guess who the real tsarevnas are, I will let you go unharmed. And if you can also figure out which one is the eldest, which one is the middle, and which one is the youngest, you can take them with you.\" Ivan can pass a note to the tsarevnas to instruct them whom to call tsarevnas. Can he, regardless of the answers of Koschei's daughters,\n\na) return alive?\n\nb) take the tsarevnas with him?\n\n#",
        "answer": "99",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56387,
        "difficulty": null
      },
      {
        "id": "9ddaafedafa493a5e8a6503e96e511847ed722db54f40edbca0d44fb1359d379",
        "domain": "numina",
        "question": "5. Given distinct complex numbers $m, n$ satisfying $m n \\neq 0$, the set $\\left\\{m^{2}, n^{2}\\right\\}=\\{m, n\\}$, then $m+n=$\n\n untranslated portion:\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nNote: The last sentence is a note about the translation request and is not part of the text to be translated. Here is the translation with the note removed:\n\n5. Given distinct complex numbers $m, n$ satisfying $m n \\neq 0$, the set $\\left\\{m^{2}, n^{2}\\right\\}=\\{m, n\\}$, then $m+n=$",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132008,
        "difficulty": null
      },
      {
        "id": "fcadc13248d314b49b52870da3c01c052244bee45b684d8428ae6f3d87e82e2e",
        "domain": "numina",
        "question": "[Examples and counterexamples. Constructions] Evaluation + example $\\quad]$\n\nOn an island, there are 100 knights and 100 liars. Each of them has at least one friend. One day, exactly 100 people said: \"All my friends are knights,\" and exactly 100 people said: \"All my friends are liars.\" What is the smallest possible number of pairs of friends, one of whom is a knight and the other a liar?",
        "answer": "50",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 55982,
        "difficulty": null
      },
      {
        "id": "bdfccbb1c35505119c168c453d1599cd1311518a7c3179362fbe1739b471abee",
        "domain": "numina",
        "question": "5. Let a regular n-gon be denoted as $A_{1} A_{2} \\ldots A_{n}$. The point $A_{3}$ is reflected over the axis $A_{2} A_{4}$, resulting in the point $A_{3}^{\\prime}$. Then, the point $A_{3}^{\\prime}$ is reflected over the axis $A_{1} A_{3}$, resulting in the point $A_{3}^{\\prime \\prime}$. For which $n \\geqq 4$ is the point $A_{3}^{\\prime \\prime}$ identical to the intersection of the lines $A_{1} A_{2}$ and $A_{3} A_{4}$?\n\n(Jaroslav Zhouf)",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75803,
        "difficulty": null
      },
      {
        "id": "4082fe6845d2ba0de68fd422902b9bb50b6cfe595a688e3eed6d72432b4dc1f4",
        "domain": "numina",
        "question": "15. (6 points) The sum of the unit digit and the tens digit of a certain two-digit number is 12. After swapping the unit digit and the tens digit, the resulting two-digit number is 36 less than the original number. The original number is $\\qquad$",
        "answer": "84",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118966,
        "difficulty": null
      },
      {
        "id": "80704e2a7efdd441ecd19249767d1c42f9ddf368f12fdcca4fa4232a3fecef81",
        "domain": "numina",
        "question": "3. A chess piece moves along the edges of a regular hexagon with a side length of $1 \\mathrm{~cm}$, starting from point $A$. On the first move, it travels $1 \\mathrm{~cm}$ to reach point $B$, on the second move, it travels $2 \\mathrm{~cm}$ to reach point $D$, $\\cdots \\cdots$, on the $n$-th move, it travels $n \\mathrm{~cm}$, until the 100th move is completed. At the point where the chess piece arrives the most times (the initial $A$ point also counts as one arrival), it has arrived $\\qquad$ times.",
        "answer": "34",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 131349,
        "difficulty": null
      },
      {
        "id": "32142b53870a7823cba1f86d5de9fd9936bf72210bd0e8b51070c29481f04399",
        "domain": "numina",
        "question": "3. A $3 \\times 3$ table needs to be filled with the numbers 2014, 2015, and 2016, such that the sum of the numbers in each row is the same. In how many different ways can this be done?",
        "answer": "831",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 6126,
        "difficulty": null
      },
      {
        "id": "675053a0b59b0f854246e5bdf9630c253de221d3813e5d8479ccf9856a6f95c2",
        "domain": "numina",
        "question": "28. (1) How many different ways are there to choose 4 letters from $a, b, b, c, c, c, d, d, d, d$? (2) How many different four-digit numbers can be formed by choosing 4 numbers from $1,2,2,3,3,3,4,4,4,4$?",
        "answer": "175",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 107236,
        "difficulty": null
      },
      {
        "id": "2d547a4666300d42df67fb928a75cd0cf4e76c8753f3ffffaa5b2f0e13faebc1",
        "domain": "numina",
        "question": "4. Given $A(0,0), B(1,0), C(0,2), D(3,3)$, point $P$ is any point in the plane, then the minimum value of $|P A|+|P B|+|P C|+|P D|$ is $\\qquad$.",
        "answer": "2\\sqrt{3}+\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 94465,
        "difficulty": null
      },
      {
        "id": "834167fee5432002364d6ee7abaa878fe43fd83a9038172a797528b05558353f",
        "domain": "numina",
        "question": "## Task 1 - 150711\n\nTwo mathematicians are discussing their different phone numbers. It turned out the following:\n\n(1) Each of the two phone numbers is a three-digit prime number.\n\n(2) Each individual digit in the two phone numbers, when considered as a single-digit number, is also a prime number.\n\n(3) The digits in the tens place of the two phone numbers are the same. The digit in the hundreds place of one phone number is the digit in the units place of the other, and vice versa.\n\nDetermine the phone numbers, and justify the result without using a table of prime numbers as evidence!",
        "answer": "337733",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 183643,
        "difficulty": null
      },
      {
        "id": "32dd8122098dd68aad60fe501c5dc56e7a8e79880a7cc4a60d9af6ce6e44ff7f",
        "domain": "numina",
        "question": "## Task 1\n\nClass 3a transferred 120 marks to the solidarity account in preparation for the 30th birthday of our republic. Class 3b transferred 98 marks. How many marks more than Class 3b did Class 3a pay into the solidarity account?",
        "answer": "22",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186645,
        "difficulty": null
      },
      {
        "id": "8135366c9774d06be55a9a842a4259ad40e75c76fc63b4efaace2eba95415e16",
        "domain": "numina",
        "question": "Four. (50 points) There are 2008 students participating in a large public welfare activity. If two students know each other, then these two students are considered as a cooperative group.\n(1) Find the minimum number of cooperative groups $m$, such that no matter how the students know each other, there exist three students who are pairwise in a cooperative group;\n(2) If the number of cooperative groups is $\\left[\\frac{m}{22}\\right]$, prove: there exist four students $A, B, C, D$ such that $A$ and $B$, $B$ and $C$, $C$ and $D$, $D$ and $A$ are each in a cooperative group.",
        "answer": "1008017",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 111046,
        "difficulty": null
      },
      {
        "id": "cdb5f9d0972f65f12272067d8c7d70c1bc0b62158977a807ab60530768f8ffaa",
        "domain": "numina",
        "question": "47. Over the past five years, 27 collective farmers have been awarded. Moreover, in each subsequent year, more were awarded than in the previous year. In the last year, three times as many were awarded as in the first year. How many collective farmers were awarded in the third year?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39356,
        "difficulty": null
      },
      {
        "id": "e7f87d841fb83ffeb62c4fd1e8a193ec4b85780623b13f0b5335ab28dedc40a2",
        "domain": "numina",
        "question": "6. Given the circle $x^{2}+y^{2}=4$ and the point $P(2,1)$, two perpendicular lines are drawn through point $P$ intersecting the circle at points $A, B$ and $C, D$ respectively, with point $A$ lying on the segment $P B$ and point $D$ lying on the segment $P C$. The maximum value of the area of quadrilateral $A B C D$ is $\\qquad$ .",
        "answer": "\\sqrt{15}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126848,
        "difficulty": null
      },
      {
        "id": "795958ce4a53bdf0dc59914538050fa23d7be9d3ddd0d69761678bb1f1cdf6d9",
        "domain": "numina",
        "question": "6. A farmer presented 6 types of sour cream in barrels of $9,13,17,19,20,38$ liters at the market. On the first day, he sold sour cream from three barrels completely, and on the second day, from two more barrels completely. The volume of sour cream sold on the first day was twice the volume of sour cream sold on the second day. Which barrels were emptied on the first day? In your answer, indicate the largest possible sum of the volumes of sour cream sold on the first day.\n\n#",
        "answer": "66",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 21799,
        "difficulty": null
      },
      {
        "id": "b2afa531607cd83c5684c4920849163445174da39091ad4d14421cc94565b55e",
        "domain": "numina",
        "question": "Given a circle with diameter $AB$ and an arbitrary point $P$ on it. The circle centered at $P$ with radius $PA$ intersects the line $AB$ at point $C$. When will the area of triangle $APC$ be the largest?",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 165386,
        "difficulty": null
      },
      {
        "id": "95a2c0aba2c600b0ce5bc2d1ec46299026add70ee503a7ff32f2da6861764b37",
        "domain": "numina",
        "question": "8. (1995 China National Team Selection Test for IMO), Let $S=\\left\\{A=\\left(a_{1}, \\cdots, a_{8}\\right) \\mid a_{i}=0\\right.$ or $\\left.1, i=1,2, \\cdots, 8\\right\\}$. For two elements $A=\\left(a_{1}, \\cdots, a_{8}\\right)$ and $B=\\left(b_{1}, \\cdots, b_{8}\\right)$ in $S$, denote $d(A, B)=\\sum_{i=1}^{8}\\left|a_{i}-b_{i}\\right|$, and call it the distance between $A$ and $B$. How many elements can be selected from $S$ at most, such that the distance between any two of them is $\\geqslant 5$?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143347,
        "difficulty": null
      },
      {
        "id": "cca892da3e9348b21eb314addacdee7dcaaaa23a27259342cabf08abef786a57",
        "domain": "numina",
        "question": "1.1. (4 points) Mitya is 11 years older than Shura. When Mitya was as old as Shura is now, he was twice as old as she was. How old is Mitya?",
        "answer": "33",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5930,
        "difficulty": null
      },
      {
        "id": "2860e6ee8556992459e4ded8e738b5a010227ab0724f1a9c1fce45e0eadff87f",
        "domain": "numina",
        "question": "80. Three students' scores in a math exam are all different, and none of them scored 0 or 100. Each student knows their own score and ranking, but not the scores or rankings of the other two. They then provide information to each other:\n甲 says: “My score is a multiple of 10.”\n乙 says: “My score is a multiple of 12.”\n丙 says: “My score is a multiple of 14.”\nAfter thinking, 乙 says: “Now I know your scores.”\nSo, 乙's score is $\\qquad$ points.",
        "answer": "84",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 108448,
        "difficulty": null
      },
      {
        "id": "0893759cf88c2a446820df40f59901ba9ae8a11785e8ed08daa04393717482ae",
        "domain": "numina",
        "question": "1. (Easy) Find the largest value of $x$ such that $\\sqrt[3]{x}+\\sqrt[3]{10-x}=1$.",
        "answer": "5+2\\sqrt{13}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 154633,
        "difficulty": null
      },
      {
        "id": "f362a562ad74c266c2624606e85a6ac8e31771bb99f43d96b92178f11bd3b465",
        "domain": "numina",
        "question": "2. [5] How many perfect squares divide $10^{10}$ ?",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151622,
        "difficulty": null
      },
      {
        "id": "150aae23bc45c3ccef1e3fbf66f849da2a3d5f43d5fb5c321edad6a7165cf699",
        "domain": "numina",
        "question": "## Task $7 / 68$\n\nWe are looking for all natural numbers $n$ for which $\\sum_{k=1}^{n} k$ is a three-digit number with identical digits.",
        "answer": "666",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182197,
        "difficulty": null
      },
      {
        "id": "e44bd20f37f3744fdb0c1464617e4f0d7b0fd7ef404b6a6edf31a53bbb370124",
        "domain": "numina",
        "question": "2.1. The vertices of a cube are labeled with numbers $\\pm 1$, and on its faces are numbers equal to the product of the numbers at the vertices of that face. Find all possible values that the sum of these 14 numbers can take. In your answer, specify their product.",
        "answer": "-20160",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5942,
        "difficulty": null
      },
      {
        "id": "3a22f000b71935abcea970215874878ac7d3da162fbd577ea8b10763967b7b38",
        "domain": "numina",
        "question": "2. Three bear cubs were dividing three pieces of cheese weighing 10 g, 12 g, and 15 g. A fox came to help them. She can simultaneously bite and eat 1 g of cheese from any two pieces. Can the fox leave the bear cubs equal pieces of cheese?",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14505,
        "difficulty": null
      },
      {
        "id": "4337c889f2fa8f966ee61aa93a188401565671d03464685999128ba79ab1255c",
        "domain": "numina",
        "question": "The equation of a circle is $x^{2}+y^{2}=100$; at the point $M_{1}(6, y>0)$ on the circumference of this circle, we draw a tangent. From the two endpoints of the diameter $EC$ lying on the x-axis, we draw perpendiculars to the tangent, intersecting it at points $A$ and $B$. What is the area of the triangle whose base is $AB$ and whose vertex is the origin $O$ of the coordinate system?",
        "answer": "80",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173214,
        "difficulty": null
      },
      {
        "id": "b1d4182985818e4bf48f26a8e5abd1622173c1ad80b1f6261c142aea6b08b9d1",
        "domain": "numina",
        "question": "2. Let $A=\\{a \\mid-1 \\leqslant a \\leqslant 2\\}$, then the area of the plane point set $B=\\{(x, y) \\mid x, y \\in A, x+y \\geqslant 0\\}$ is $\\qquad$ .",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 117103,
        "difficulty": null
      },
      {
        "id": "1c5feb5490dcd5b9764adef0ac2acb026f0818693a604c44f45c3664bebf1022",
        "domain": "numina",
        "question": "90. In a sequence of integers (positive), each term, starting from the third, is equal to the absolute difference of the two preceding ones.\n\nWhat is the maximum number of terms such a sequence can have if each of its terms does not exceed 1967?",
        "answer": "2952",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 25295,
        "difficulty": null
      },
      {
        "id": "0739e0586f6dc3b36b74142f8de352d677c415cb7d22f4dde9a5d110b8077e6a",
        "domain": "numina",
        "question": "8. (6 points) Aunt Zhang and Aunt Li have the same monthly salary. Aunt Zhang deposits 30% of her salary into the bank each month, and the rest is used for daily expenses. Aunt Li's monthly daily expenses are 10% more than Aunt Zhang's, and the remaining money is also deposited into the bank. After one year, Aunt Li found that the total amount she deposited into the bank over 12 months was 5880 yuan less than Aunt Zhang's. What is Aunt Li's monthly salary? $\\qquad$",
        "answer": "7000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125394,
        "difficulty": null
      },
      {
        "id": "722c05572a3752de6e4da82ec437bedbd4c890ec5a32dc7a2cf51d5a4efe6a06",
        "domain": "numina",
        "question": "447. Find the greatest common divisor of the numbers 111111 and 111111111.",
        "answer": "111",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 45178,
        "difficulty": null
      },
      {
        "id": "c753297f4fd1a29cf3e1159b3610009e6d904eb276b0b748ccf0184a717ef125",
        "domain": "numina",
        "question": "Problem 8. In triangle $A B C$, side $B C=28$. The bisector $B L$ is divided by the point of intersection of the bisectors of the triangle in the ratio $4: 3$, counting from the vertex. Find the radius of the circumscribed circle around triangle $A B C$, if the radius of the inscribed circle in it is 12.",
        "answer": "50",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9422,
        "difficulty": null
      },
      {
        "id": "7622573c5b9208ff3aefe40383e53c779f55cf693f7a85950d7ab1f7dcc32c8d",
        "domain": "numina",
        "question": "In the following figure, the angles marked in gray have the same measure. Similarly, the angles marked in white also have the same measure. Determine the measure of angle $b$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_27f67f4520ebb8354fb2g-11.jpg?height=434&width=831&top_left_y=548&top_left_x=710)\n\n#",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73240,
        "difficulty": null
      },
      {
        "id": "2639fb4dd2f8793b25ed06ee1c31dff1da52c292c35c1388ede96338017e3971",
        "domain": "numina",
        "question": "10. The number of triangles with integer sides and the longest side being 11 is $\\qquad$.\n\nThe total number of such triangles is $\\qquad$.",
        "answer": "36",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104828,
        "difficulty": null
      },
      {
        "id": "f8eeba64f5e1b8e225ecb4e3bfd718eed84d67568bd60eaea861e9f516e2a840",
        "domain": "numina",
        "question": "9. Given a cube $A B C D-A_{1} B_{1} C_{1} D_{1}$ with edge length 1, a sphere is constructed with vertex $A$ as the center and $\\frac{2 \\sqrt{3}}{3}$ as the radius. The length of the curve formed by the intersection of the sphere's surface with the cube's surface is $\\qquad$.",
        "answer": "\\frac{5\\sqrt{3}\\pi}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92179,
        "difficulty": null
      },
      {
        "id": "5dcb9676d2d97a2c991a21c793879300b5773d12dc84c9ba8f68265e04885a47",
        "domain": "numina",
        "question": "Marayev A.\n\nCubes of size $1 \\times 1 \\times 1$ are glued together to form a cube of size $3 \\times 3 \\times 3$. What is the maximum number of cubes that can be removed from it so that the remaining figure has the following two properties:\n\n- from the side of each face of the original cube, the figure looks like a $3 \\times 3$ square (looking perpendicularly at this face, we do not see any gaps - 9 cubes of the figure are visible);\n- by moving from cube to cube through their common face, one can get from any cube to any other within the figure?",
        "answer": "13",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 52643,
        "difficulty": null
      },
      {
        "id": "b1cb770428b8335ad6c21b646be7e194e2e452210449a4b4f65779b81407b83c",
        "domain": "numina",
        "question": "A A trapezoid is divided into seven strips of equal width as shown. What fraction of the trapezoid's area is shaded? Explain why your answer is correct.",
        "answer": "\\frac{4}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 123553,
        "difficulty": null
      },
      {
        "id": "60739c0c3f213e9b3486223261d002b641d964a8d65497ab7e557d77f4e7694a",
        "domain": "numina",
        "question": "1. Calculate:\n$$\n\\frac{\\frac{7}{18} \\times 4 \\frac{1}{2}+\\frac{1}{6}}{13 \\frac{1}{3}-3 \\frac{3}{4} \\div \\frac{5}{16}} \\times 2 \\frac{7}{8}\n$$",
        "answer": "4\\frac{17}{128}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92030,
        "difficulty": null
      },
      {
        "id": "dd4b7570cebaa915d731c4d6bf2a85b7c2bf8696b04def88ff69ac9489e9f9cc",
        "domain": "numina",
        "question": "5. A supermarket is holding a promotional event, offering two packages in the discount area. Package $\\mathrm{A}$ includes 2 bottles of milk, and Package $\\mathrm{B}$ includes 3 bottles of cola. Each package comes with a free souvenir. After purchasing both types of packages, Xiao Long found that he bought a total of 50 bottles of milk and cola, and received 22 souvenirs. How many bottles of milk did Xiao Long buy?",
        "answer": "32",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94842,
        "difficulty": null
      },
      {
        "id": "e29d6a64918934c35ca6fea0841e3142bafab833a186c9f63a6c448d20e6a371",
        "domain": "numina",
        "question": "9. (16 points) In the acute triangle $\\triangle ABC$, $a, b, c$ are the sides opposite to $\\angle A, \\angle B, \\angle C$ respectively, and $a, b, c$ form an arithmetic sequence, $\\sin (A-C)=\\frac{\\sqrt{3}}{2}$. Find $\\sin (A+C)$.",
        "answer": "\\frac{\\sqrt{39}}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104833,
        "difficulty": null
      },
      {
        "id": "e729caeca72774181d9aa161b862b40b43ce4c60e5764616c5f35e6f9ac645a0",
        "domain": "numina",
        "question": "Šárka enjoys celebrating, so in addition to birthdays, she came up with antibirthdays: the date of the antibirthday is created by swapping the day and month numbers in the birth date. She was born on 8.11., so her antibirthday is on 11.8. Her mother cannot celebrate her antibirthday: she was born on 23.7., her antibirthday would be on 7.23., which is not a valid date. Her brother can celebrate his antibirthday, but it falls on the same day as his birthday: he was born on 3.3.\n\nHow many days in a year are such that a person born on that day can celebrate their antibirthday, and it is on a different day than their birthday?\n\n(V. Hucíková)\n\nHint. What day of the month must a person be born on to be able to celebrate their antibirthday?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\nŠárka enjoys celebrating, so in addition to birthdays, she came up with antibirthdays: the date of the antibirthday is created by swapping the day and month numbers in the birth date. She was born on 8.11., so her antibirthday is on 11.8. Her mother cannot celebrate her antibirthday: she was born on 23.7., her antibirthday would be on 7.23., which is not a valid date. Her brother can celebrate his antibirthday, but it falls on the same day as his birthday: he was born on 3.3.\n\nHow many days in a year are such that a person born on that day can celebrate their antibirthday, and it is on a different day than their birthday?\n\n(V. Hucíková)\n\nHint. What day of the month must a person be born on to be able to celebrate their antibirthday?",
        "answer": "132",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76702,
        "difficulty": null
      },
      {
        "id": "9f7982c6ab5b25de1f599ee842f6a259dfad2aaa390ff72e05c4b300c76ec3e6",
        "domain": "numina",
        "question": "7. Let the edge length of a regular tetrahedron be $2 \\sqrt{6}$, and let a sphere be constructed with the center $O$ of the tetrahedron as its center. The total length of the curves formed by the intersection of the sphere's surface with the four faces of the tetrahedron is $4 \\pi$. Then the radius of the sphere $O$ is $\\qquad$",
        "answer": "\\frac{\\sqrt{5}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 115246,
        "difficulty": null
      },
      {
        "id": "20d27a16446cd93748c3f7191e899819c7a40639d6b2732d279ae982d0bc1097",
        "domain": "numina",
        "question": "[ Inequalities. Case Analysis ]\n\nA magical moment is when the number of minutes on a digital clock matches the number of hours. To brew a magical potion, it must be put on the fire and taken off the fire at magical moments. To make it taste good, it must be cooked for 1.5 to 2 hours. How long does it take to cook a delicious magical potion?\n\n#",
        "answer": "1",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 70477,
        "difficulty": null
      },
      {
        "id": "1ebf0f3c2df098f6eed2e4641b9f575a7969d20bea275ff9227eb710b4f90669",
        "domain": "numina",
        "question": "5.1. Let's consider the number 616. The sum of its digits is 13, and the product of its digits is 36. What is the largest natural number whose sum of digits is 13 and the product of its digits is 36?",
        "answer": "3322111",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14141,
        "difficulty": null
      },
      {
        "id": "ded4c1a740cd3d48a4a9aba08cd15477af28bd0136cdca46a64f3da247e58b2a",
        "domain": "numina",
        "question": "$11 \\cdot 30 \\quad N$ is an integer, its $b$-ary representation is 777. Find the smallest positive integer $b$, such that $N$ is the fourth power of an integer in decimal.\n(9th Canadian Mathematics Competition, 1977)",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 127555,
        "difficulty": null
      },
      {
        "id": "8e49400655c35e7e08c6bba79716a9baff12b73b534bb31feb9cee40af773596",
        "domain": "numina",
        "question": "10.3 For the function $f(x)=2013+\\sin 2 \\pi x-8 x^{3}-12 x^{2}-18 x-a$, find the number of integer values of $a$ for each of which the equation\n\n$$\n\\underbrace{f(f(\\ldots f}_{2013 \\text { times }}(x) \\ldots))=2 x+1\n$$\n\nhas a unique solution on the interval $[49,50]$.",
        "answer": "60017",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4972,
        "difficulty": null
      },
      {
        "id": "dc4ac249bebe6f763dd0bef81b9eb5364a79cb66d6ee74d86f6ee0b0d29cba36",
        "domain": "numina",
        "question": "5. The sum of 10 positive integers is 24, the maximum value of the sum of their squares is $a$, and the minimum value is $b$, then $a +$ $b=$ . $\\qquad$",
        "answer": "294",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144673,
        "difficulty": null
      },
      {
        "id": "d5528709fe0f47eaa443d9320243ab8fe8436bc7ea9128a0bc65df1e73814a7e",
        "domain": "numina",
        "question": "10.5. Find all natural numbers $\\boldsymbol{n}$ such that $\\boldsymbol{n}$ is equal to the sum of three numbers, the first of which is the largest divisor of the number $\\boldsymbol{n} \\boldsymbol{- 1}$, different from $\\boldsymbol{n}-1$, the second is the largest divisor of the number $\\boldsymbol{n}-\\mathbf{2}$, different from $\\boldsymbol{n}-\\mathbf{2}$, and the third is the largest divisor of the number $\\boldsymbol{n}-\\mathbf{3}$, different from $\\boldsymbol{n}-\\mathbf{3}$.",
        "answer": "5866",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9733,
        "difficulty": null
      },
      {
        "id": "56141f3ae69a2589a2efb64ae59b66d995d03de8d82a2ee401f41b03a38e672e",
        "domain": "numina",
        "question": "10. Let $a, b, c \\in(0,1]$, $\\lambda$ be a real number, such that $\\frac{\\sqrt{3}}{\\sqrt{a+b+c}} \\geqslant 1+\\lambda(1-a)(1-b)(1-c)$ always holds, find the maximum value of $\\lambda$.\n\nSet $a, b, c \\in(0,1]$, $\\lambda$ as a real number, such that $\\frac{\\sqrt{3}}{\\sqrt{a+b+c}} \\geqslant 1+\\lambda(1-a)(1-b)(1-c)$ always holds, find the maximum value of $\\lambda$.",
        "answer": "\\frac{64}{27}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 97706,
        "difficulty": null
      },
      {
        "id": "ececf27e775a8cc090c080d8151705e1e7525a3f31dfa961f5b0989a8f3c0692",
        "domain": "numina",
        "question": "[ Skeletons of Polyhedral Figures ]\n\nA toy factory produces wireframe cubes, with small colorful beads located at their vertices. According to the GOST standard, each cube must use beads of all eight colors (white and the seven colors of the rainbow). How many different models of cubes can the factory produce?\n\n#",
        "answer": "1680",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57678,
        "difficulty": null
      },
      {
        "id": "a640446b21a54faa15fb52bbc97045456ec0af76e4396dfa517317c60cff0fd9",
        "domain": "numina",
        "question": "Problem 6. A square table of size $7 \\times 7$ with the four corner squares deleted is given.\n\nFigure 4.\n\na) What is the smallest number of squares which need to be colored black so that a 5 -square entirely uncolored Greek cross (Figure 4) cannot be found\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_1ba58ed6da37450ce759g-4.jpg?height=211&width=212&top_left_y=1008&top_left_x=448)\non the table?\n\nb) Prove that it is possible to writte integers in each square in a way that the sum of the integers in each Greek cross is negative while the sum of all integers in the square table is positive.",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 176941,
        "difficulty": null
      },
      {
        "id": "09d682a36be60d1782ce75d91eacac1585cdf55342c911163d4a650ced7bde6a",
        "domain": "numina",
        "question": "10.3. Find all natural $k$ such that the product of the first $k$ prime numbers, decreased by 1, is a perfect power of a natural number (greater than the first).\n\n(V. Senderov)",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12214,
        "difficulty": null
      },
      {
        "id": "180433bcfd7d206c355edf557ae80f2cedaf52dc51526eda17b096c52d1a207b",
        "domain": "numina",
        "question": "Problem 11.1. The sequence $\\left\\{a_{n}\\right\\}_{n=1}^{\\infty}$ is defined by $a_{1}=0$ and $a_{n+1}=$ $a_{n}+4 n+3, n \\geq 1$.\n\na) Express $a_{n}$ as a function of $n$.\n\nb) Find the limit\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{\\sqrt{a_{n}}+\\sqrt{a_{4 n}}+\\sqrt{a_{4^{2} n}}+\\cdots+\\sqrt{a_{4}{ }^{10} n}}{\\sqrt{a_{n}}+\\sqrt{a_{2 n}}+\\sqrt{a_{2^{2} n}}+\\cdots+\\sqrt{a_{2^{10} n}}}\n$$\n\nEmil Kolev",
        "answer": "683",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 177615,
        "difficulty": null
      },
      {
        "id": "07f791a9af4f68294a6d14264b6f58bdf0c7a8381b9651139730263218cec6b2",
        "domain": "numina",
        "question": "1. In some cells of a $1 \\times 2100$ strip, one chip is placed. In each of the empty cells, a number is written that is equal to the absolute difference between the number of chips to the left and to the right of this cell. It is known that all the written numbers are distinct and non-zero. What is the minimum number of chips that can be placed in the cells?",
        "answer": "1400",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7741,
        "difficulty": null
      },
      {
        "id": "7ab2b53bdd000a5907d917e33a9962b10b551e10d3bad9959c55820368219451",
        "domain": "numina",
        "question": "1. At Andrei's birthday, Yana was the last to arrive, giving him a ball, and Eduard was the second to last, giving him a calculator. Testing the calculator, Andrei noticed that the product of the total number of his gifts and the number of gifts he had before Eduard arrived is exactly 16 more than the product of his age and the number of gifts he had before Yana arrived. How many gifts does Andrei have?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6306,
        "difficulty": null
      },
      {
        "id": "2ea82c7bcef0a380e4260817ab5037620167482589d56006882c69c1b2073781",
        "domain": "numina",
        "question": "[ Periodicity and Aperiodicity $]$ [ Classical Combinatorics (other)]\n\nIn an old manor, the house is surrounded by tall trees - pines, spruces, and birches. There are a total of 96 trees. These trees have a strange property: of the two trees growing two apart from any conifer, one is coniferous and the other is deciduous, and of the two trees growing four apart from any conifer, one is coniferous and the other is deciduous. How many birches are planted around the house?",
        "answer": "32",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 68898,
        "difficulty": null
      },
      {
        "id": "38e56ee1b8c174e1f52532bd9c749105b7d054e3f3b1eb70ecc88c43b773133e",
        "domain": "numina",
        "question": "$1.53 \\frac{\\sqrt{7+4 \\sqrt{3}} \\cdot \\sqrt{19-8 \\sqrt{3}}}{4-\\sqrt{3}}-\\sqrt{3}=2$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49491,
        "difficulty": null
      },
      {
        "id": "ee52e56816beb56c954da0086c98c0b9baca3872ebf54cdd63f5063ee0dea583",
        "domain": "numina",
        "question": "4. The board for playing \"Battleship\" is $10 \\times 10$. What is the maximum number of ships of size $1 \\times 4$ that can be placed on it?",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5104,
        "difficulty": null
      },
      {
        "id": "a266e12f882af2a9c185c76e8dd8f2e0f7a652f8cc5037656f9d3135b8ff8d3d",
        "domain": "numina",
        "question": "Let's determine the value of the following number to four decimal places:\n\n$$\n2+\\sqrt{3+\\sqrt{5-\\sqrt{13+\\sqrt{48}}}}\n$$",
        "answer": "3.9319",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 165056,
        "difficulty": null
      },
      {
        "id": "71874743fefee554516cb18d2ba487f1309831ff6b9158ebbc853afc93228a3f",
        "domain": "numina",
        "question": "Example 8. At a distribution base, there are electric bulbs manufactured by two factories. Among them, $60 \\%$ are made by the first factory and $40 \\%$ by the second. It is known that out of every 100 bulbs made by the first factory, 95 meet the standard, and out of 100 bulbs made by the second factory, 85 meet the standard. Determine the probability that a randomly selected bulb will meet the standard.",
        "answer": "0.91",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32971,
        "difficulty": null
      },
      {
        "id": "2db964b37c4240afd68b03c82ed96c55be0e6f9b07f639f2ddc0033475799c89",
        "domain": "numina",
        "question": "[ Properties and characteristics of an isosceles triangle. ]\n\n[ Pythagorean Theorem (direct and inverse). ]\n\nThe lateral sides of a right trapezoid are 10 and 8. The diagonal of the trapezoid, drawn from the vertex of the acute angle, bisects this angle.\n\nFind the area of the trapezoid.",
        "answer": "104",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64762,
        "difficulty": null
      },
      {
        "id": "c4a3bb4fce0701a92e039b246eb31f0c683d5afef7c1dbc036a26a9ceec1194a",
        "domain": "numina",
        "question": "4. The length of the escalator is 200 steps. When Petya walks down the escalator, he manages to count 50 steps. How many steps will he count if he runs twice as fast?",
        "answer": "80",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 21550,
        "difficulty": null
      },
      {
        "id": "dd8a9cc1033fda75492dd4cc473448a362a00e1276c50fbe3705897ef58d8e90",
        "domain": "numina",
        "question": "Let $M=\\{1,2,3, \\cdots, 1995\\}, A$ be a subset of $M$ and satisfy the condition: when $x \\in A$, $15x \\notin A$. Then the maximum number of elements in $A$ is $\\qquad$ .",
        "answer": "1870",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 128352,
        "difficulty": null
      },
      {
        "id": "2075615c690a69c84912e363080d9db8eb68836d78085aed3d1068e131763bd1",
        "domain": "numina",
        "question": "## Task 2 - 240732\n\na) Let $M$ be the set of all numbers $x$ that have the following properties (1), (2), (3):\n\n(1) $x$ is a six-digit natural number.\n\n(2) $x$ has the digit sum 29.\n\n(3) $x$ is divisible by 11.\n\nDetermine the largest element of the set $M$!\n\nb) Let $M^{\\prime}$ be the set of all numbers $x$ that, in addition to properties (1), (2), (3), also have the following property (4):\n\n(4) No two digits of $x$ are the same.\n\nDetermine the largest element of the set $M^{\\prime}$!",
        "answer": "999020",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 183758,
        "difficulty": null
      },
      {
        "id": "3462bd5f0633762bb2d72f930386f23dc6c231b58be31e8e9b195272c15e3aff",
        "domain": "numina",
        "question": "# 8.\n\nThe answer to the task should be some integer or a number written as a finite decimal. If the answer contains a fractional number, use a comma when writing it. Enter all characters (comma, digits) without spaces. There is no need to indicate units of measurement.\n\nThe base and the side of the triangle are 30 and 14, respectively. Find the area of this triangle if the median drawn to the base is 13.",
        "answer": "168",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19267,
        "difficulty": null
      },
      {
        "id": "aac43d5211daef5983ffa6321f472cdd7bd31e521780a58ff20e5e4e2caff971",
        "domain": "numina",
        "question": "7. Let $\\triangle A B C$ be a right triangle with right angle $C$. Let $I$ be the incenter of $A B C$, and let $M$ lie on $A C$ and $N$ on $B C$, respectively, such that $M, I, N$ are collinear and $\\overline{M N}$ is parallel to $A B$. If $A B=36$ and the perimeter of $C M N$ is 48 , find the area of $A B C$.",
        "answer": "252",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151581,
        "difficulty": null
      },
      {
        "id": "d99ea5f0ef2531274b343811d76155d2663c4898be53d57cb462c066a7d3ad9e",
        "domain": "numina",
        "question": "Berov s.L.\n\nSeryozha chose two different natural numbers $a$ and $b$. He wrote down four numbers in his notebook: $a, a+2, b$ and $b+2$. Then he wrote on the board all six pairwise products of the numbers from the notebook. What is the maximum number of perfect squares that can be among the numbers on the board?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 55550,
        "difficulty": null
      },
      {
        "id": "e6b7f49573f538e6036ab190d998c30f3609ae08de4182b1e60dcfd318021b2e",
        "domain": "numina",
        "question": "9. As shown in the figure, the area of trapezoid $A B C D$ is 117 square centimeters. $A D / / B C$, $E F=13$ centimeters, $M N=4$ centimeters, and it is also known that $E F \\perp M N$ at $O$. What is the total area of the shaded parts? $\\qquad$ square centimeters.",
        "answer": "65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126258,
        "difficulty": null
      },
      {
        "id": "937f1a4540417a50cc1318d7c30ccf326287b10cafbb857bcfbb7d4e3acdd50d",
        "domain": "numina",
        "question": "In the diagram, the circle has centre $O$ and radius 6 . Point $A$ is outside the circle and points $B$ and $C$ are on the circle so that $A B$ is perpendicular to $B O, A C$ is perpendicular to $C O$, and $\\angle B A C=50^{\\circ}$. What is the area of the shaded region?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2efeb2001664c4e9869eg-2.jpg?height=396&width=572&top_left_y=325&top_left_x=836)",
        "answer": "13\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194289,
        "difficulty": null
      },
      {
        "id": "04ecc235de2af713098809ad53111a60d4e11813cc02118bdd541909a158674b",
        "domain": "numina",
        "question": "Tokarev S.I.\n\nThe path from platform $A$ to platform $B$ was traveled by the electric train in $X$ minutes ($0<X<60$). Find $X$, given that the angle between the hour and minute hands was $X$ degrees both at the moment of departure from $A$ and at the moment of arrival at $B$.",
        "answer": "48",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 72097,
        "difficulty": null
      },
      {
        "id": "cefc0d7016b017d4a60cbf7ffb639aa852fd44da08c1e481d366da9cd8fab400",
        "domain": "numina",
        "question": "68. Nail two boards of the same length together, resulting in a board that is 39 centimeters long. The overlapping part in the middle is 11 centimeters long. Each of these two boards is $\\qquad$ centimeters long.",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102310,
        "difficulty": null
      },
      {
        "id": "a329c4db1a9903554ce1c56caae3affa41ae818f5a68e6c31dc46b9770324fbe",
        "domain": "numina",
        "question": "37. Football teams were holding a round-robin tournament. One of the fans of this game, returning from vacation, found the following situation: the total number of points scored by all teams was 44; one team, which scored the fewest points, received one point; the two teams occupying the first and second places scored the same number of points. How many teams are participating in the tournament and how many games do they still have to play?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 39737,
        "difficulty": null
      },
      {
        "id": "bf8865f44e9c246147585b72e2d3c2f33d27934c110ca858eb979b221676d087",
        "domain": "numina",
        "question": "16. Let $f(x)=\\frac{x^{2010}}{x^{2010}+(1-x)^{2010}}$. Find the value of\n$$\nf\\left(\\frac{1}{2011}\\right)+f\\left(\\frac{2}{2011}\\right)+f\\left(\\frac{3}{2011}\\right)+\\ldots+f\\left(\\frac{2010}{2011}\\right)\n$$",
        "answer": "1005",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160598,
        "difficulty": null
      },
      {
        "id": "d20a8da4a14d3935c4edd6a08254c0e1392b68a0805b726a5c3241a37aca5a82",
        "domain": "numina",
        "question": "32. It is given that $p$ is a prime number such that\n$$\nx^{3}+y^{3}-3 x y=p-1\n$$\nfor some positive integers $x$ and $y$. Determine the largest possible value of $p$.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160725,
        "difficulty": null
      },
      {
        "id": "3896196d1418cbc3a3e3d582cb91259e1568b03099b944c046d21ff8ec1043ed",
        "domain": "numina",
        "question": "3. Solve the equation\n\n$$\n2(x-6)=\\frac{x^{2}}{(1+\\sqrt{x+1})^{2}}\n$$",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19299,
        "difficulty": null
      },
      {
        "id": "41bf17753ad597c81a37bc5b9dfdd67c32ef0e3213d43f26b7a49bb76cbc3e3b",
        "domain": "numina",
        "question": "14.5. a) Seven girls are forming a circle. In how many different ways can they stand in a circle?\n\nb) How many necklaces can be made from seven different beads?",
        "answer": "360",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 38513,
        "difficulty": null
      },
      {
        "id": "7e491bbe69b37a152d63ce84c60d8cb9968034e648234805ade5647f2071faca",
        "domain": "numina",
        "question": "## Task A-3.2. (4 points)\n\nLegoplus is a body consisting of seven equal cubes joined in such a way that there is one cube that shares a common face with each of the remaining six cubes.\n\nEach face of the legoplus must be painted with one color. How many colors are minimally needed to do this so that no two adjacent faces are painted the same color?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88333,
        "difficulty": null
      },
      {
        "id": "dd5be0618c5bc334159ecde9ad31e2bfa4fa32871652df50e1bc7e309d03de3a",
        "domain": "numina",
        "question": "G2.3 Given that there are $c$ prime numbers less than 100 such that their unit digits are not square numbers, find the values of $c$.",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153010,
        "difficulty": null
      },
      {
        "id": "d8e7dc17be20415fda3ab850f047f9b68ab44d4a4f573f4a145ca54fe190ffae",
        "domain": "numina",
        "question": "1. How many weeks can a year overlap? Assume that a year overlaps with a week if at least 6 and one day of this week falls within the given year.",
        "answer": "54",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 5795,
        "difficulty": null
      },
      {
        "id": "750992bc5346c86ad65935fe4f121ae8c372482b4a4958d6e4f4914df2202387",
        "domain": "numina",
        "question": "Task B-3.1. Solve the equation\n\n$$\n\\left(4-\\frac{x}{2013}\\right)^{10^{2013}}=\\left(\\frac{x}{671}\\right)^{10^{2013}}\n$$",
        "answer": "-40262013",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88494,
        "difficulty": null
      },
      {
        "id": "11f2756f35b162815a9ff46146a132c4792661b1c8b85e0832e173180eabd87d",
        "domain": "numina",
        "question": "G10.3The fraction $\\frac{p}{q}$ is in its simplest form. If $\\frac{7}{10}<\\frac{p}{q}<\\frac{11}{15}$ where $q$ is the smallest possible positive integer and $c=p q$. Find the value of $c$.",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152785,
        "difficulty": null
      },
      {
        "id": "6107367dd9d91b135f395aa153bc3f0b5c3e8619f9a5ab85b4f501e6ca2ec282",
        "domain": "numina",
        "question": "## Task 1\n\nIn front of the circus tent, students from a first grade class are standing.\n\nThe children have lined up in pairs. The teacher gives out 2 admission tickets at a time. She distributes four times 2 tickets to the girls and five times 2 tickets to the boys.\n\nQuestions:\n\na) How many girls want to go to the circus?\n\nb) How many boys are standing in front of the circus?\n\nc) How many children receive an admission ticket from the teacher?",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186298,
        "difficulty": null
      },
      {
        "id": "837eb2a0f866ae450aab2096376f5044d186e483f7bc62c6a3ec177213b2c08d",
        "domain": "numina",
        "question": "For any positive integer $n$, let $f_{1}(n)$ denote the square of the sum of the digits of $n$ plus $r+1$, where $r$ is the integer such that $n=3q+r, 0 \\leqslant r<3$. For $k \\geqslant 2$, let $f_{k}(n)=f_{1}\\left(f_{k-1}(n)\\right)$. Therefore, $f_{1990}(2345)$ $=$ $\\qquad$.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103891,
        "difficulty": null
      },
      {
        "id": "b37b569fb423a16b4bc88f5e62db1539eee1a95fc6ad7687498f23f34e1388ff",
        "domain": "numina",
        "question": "5. In the lottery box of Green Green Park, there are golden, silver, and colorful balls. The lottery participant can randomly draw 3 balls from the lottery box. Xiao Yangyang hopes to get 3 golden balls, Mei Yangyang hopes to get 2 golden balls and 1 silver ball, and Lan Yangyang hopes to get one of each kind of ball. Village Chief Man Yangyang says: “Only one of you three can be chosen to draw the lottery, no matter who is chosen, the probability of achieving your wish is the same.” There are at least $\\qquad$ balls in the lottery box.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 109112,
        "difficulty": null
      },
      {
        "id": "bcf76b9d1b283692f4b0acc48c9bf0c3558304e7b768fcab8a15117cca7fc592",
        "domain": "numina",
        "question": "13.288. A master and his apprentice were tasked with manufacturing a batch of identical parts. After the master worked for 7 hours and the apprentice for 4 hours, it turned out that they had completed $5 / 9$ of the entire work. After working together for another 4 hours, they found that $1 / 18$ of the entire work remained to be done. How long would it take the apprentice to complete the entire work on his own?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49374,
        "difficulty": null
      },
      {
        "id": "869a8a1af30cd445e28ae08f9738d9585bbd2d8a12b2f9760d6e3a36408407d4",
        "domain": "numina",
        "question": "6. How many solutions does the rebus $\\frac{B+O+C+b+M+O+\\breve{U}}{K+J+A+C+C}=\\frac{22}{29}$ have, where different letters represent different digits, and the same letters represent the same digits? It is known that the digit 0 is not used.",
        "answer": "0",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13294,
        "difficulty": null
      },
      {
        "id": "370abc06aae70f6dc69c8c055eee1f05bdb14e62e3e97889944d626d290e909c",
        "domain": "numina",
        "question": "11. Andrew, Bob, and Chris are working together to finish a group project. If Andrew doesn't help, it would take 2 hours. If Bob doesn't help, it would take 3 hours. If Chris doesn't help, it would take 4 hours. Now they just learned that Dave can also help them out. If Dave works on the project alone, it would take him half a day. What is the least amount of time it would take for them to finish the group project?",
        "answer": "\\frac{8}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 154647,
        "difficulty": null
      },
      {
        "id": "f5a35ad28d6d202fac1e9527f965b3f43b958e0f22c12f8139fa759d5a30d545",
        "domain": "numina",
        "question": "3. Determine the smallest and largest value of the function\n\n$$\nf(x, y, z, t)=\\frac{a x^{2}+b y^{2}}{a x+b y}+\\frac{a z^{2}+b t^{2}}{a z+b t},(a>0, b>0)\n$$\n\nunder the conditions $x+z=y+t=1, x, y, z, t \\geq 0$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189665,
        "difficulty": null
      },
      {
        "id": "4296c1878e0d5a9620350acccf280af849b69cbff90d0e98a7c9790442ea8c02",
        "domain": "numina",
        "question": "Let's determine the $x$ for which the\n\n$$\n|13-x|+|58-x|+|71-x|+|75-x|+|79-x|\n$$\n\nsum is minimal!",
        "answer": "71",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167471,
        "difficulty": null
      },
      {
        "id": "a49fb9ff37f6ae4a0467418e0b765b64c4fa3e7e7b74e1f9dc43507dd162477e",
        "domain": "numina",
        "question": "[ Inscribed and described circles ]\n\nIn triangle $A B C$ with sides $A B=\\sqrt{3}, B C=4, A C=\\sqrt{7}$, the median $B D$ is drawn. The circles inscribed in triangles $A B D$ and $B D C$ touch $B D$ at points $M$ and $N$ respectively. Find $M N$.\n\n#",
        "answer": "2-\\frac{\\sqrt{3}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67031,
        "difficulty": null
      },
      {
        "id": "4ad356e5501186e975f067824a35e64e5a72b47065ea4ca3fd804badf3d1726b",
        "domain": "numina",
        "question": "Problem 4.6.3 A rectangle with a perimeter of 96 cm was divided into 108 smaller identical rectangles by eight vertical and eleven horizontal cuts. What is the perimeter of each of them, if the total length of all the cuts is 438 cm?",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17223,
        "difficulty": null
      },
      {
        "id": "0c54f28bb01300c32617b29f5902b209417d8b2a1b9a30166cc8a447bba76ca4",
        "domain": "numina",
        "question": "An equilateral triangle has sides of length $x+5, y+11$, and 14 . What is the value of $x+y$ ?\n\n##",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194115,
        "difficulty": null
      },
      {
        "id": "b9ad183604b56358e1d323f9d9f2ea75c58d38a97069016d59f0b104161ce076",
        "domain": "numina",
        "question": "1. In a chess tournament, 10 players participated. Each player played one game with each of the other players.\n\na) How many games did each individual player play?\n\nb) How many games of chess were played in total at the tournament?\n\nExplain your answers.",
        "answer": "45",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86904,
        "difficulty": null
      },
      {
        "id": "76052a793dd8a22a7e49f17d532a2bf982b92cffc487e3b96429bb71bac65593",
        "domain": "numina",
        "question": "2. Let $S$ be a subset of $\\{1,2, \\ldots, 9\\}$ such that the sums of any two elements of $S$ are distinct. For example, the set $\\{1,2,3,5\\}$ has this property, but the set $\\{1,2,3,4,5\\}$ does not, because $\\{2,3\\}$ and $\\{1,4\\}$ both sum to 5. What is the maximum number of elements that $S$ can contain?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 188475,
        "difficulty": null
      },
      {
        "id": "a69679101bbd4057e5c180cc625e89f4b14b5616e8b05bd72d76565b4b275ecf",
        "domain": "numina",
        "question": "23. In $\\triangle A B C, A B: A C=4: 3$ and $M$ is the midpoint of $B C$. $E$ is a point on $A B$ and $F$ is a point on $A C$ such that $A E: A F=2: 1$. It is also given that $E F$ and $A M$ intersect at $G$ with $G F=72 \\mathrm{~cm}$ and $G E=x \\mathrm{~cm}$. Find the value of $x$.",
        "answer": "108",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160054,
        "difficulty": null
      },
      {
        "id": "14ae5b9120acafbda679eb8306d507be0841c8d22f234e0ebb6d27e5eb34b139",
        "domain": "numina",
        "question": "# Task 3. (3 points)\n\nIn the country of Taxia, everyone pays as many percent in taxes as thousands of tugriks their salary amounts to. What salary is the most advantageous to have?\n\n(Salary is measured in a positive, not necessarily integer number of tugriks)",
        "answer": "50000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 20374,
        "difficulty": null
      },
      {
        "id": "b03ef5d4a3eb6ecc4044458be4cd19f5756afdb5932fd487e2d440f93d83bc21",
        "domain": "numina",
        "question": "7. Find the value of\n$$\n\\sum_{a=1}^{\\infty} \\sum_{b=1}^{\\infty} \\sum_{c=1}^{\\infty} \\frac{a b(3 a+c)}{4^{a+b+c}(a+b)(b+c)(c+a)}\n$$",
        "answer": "\\frac{1}{54}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151769,
        "difficulty": null
      },
      {
        "id": "28a1892d7fa8371cf528bc4f04b61c6f67412529263ff95dc961cdfe577b5ca9",
        "domain": "numina",
        "question": "3. (20 points) In math class, each dwarf needs to find a three-digit number without zero digits, divisible by 3, such that when 297 is added to it, the result is a number with the same digits but in reverse order. What is the minimum number of dwarfs that should be in the class so that among the numbers they find, there are always at least two identical ones?",
        "answer": "19",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 7838,
        "difficulty": null
      },
      {
        "id": "cd0960681bdad38fa6df424519ccf108cfe0c7e2658eb8c98b2f640bbd2b6662",
        "domain": "numina",
        "question": "[Example 3.2.6] Find a three-digit number such that the ratio of the number to the sum of its digits is minimized.",
        "answer": "199",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 97041,
        "difficulty": null
      },
      {
        "id": "956e72a9ae84ef01bfbacf1a04d6e9530588c580688489d0847f869663fbc984",
        "domain": "numina",
        "question": "【Question 2】Using 1 one, 2 twos, 2 threes to form some four-digit numbers, the total number of different four-digit numbers that can be formed is $\\qquad$.\n\nTranslating the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "30",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143020,
        "difficulty": null
      },
      {
        "id": "f684ee8fa1d352418305329b04c04379e4417c33bce885d73f7fe23132cd4bb5",
        "domain": "numina",
        "question": "8. When we play poker, if we get 2 cards of the same size (such as 2 fives), we say we have a \"pair of fives.\" If we get three cards of the same size (such as 3 Ks), we say we have a \"trio of Ks.\" If we get 4 cards of the same size, we say we have a \"bomb.\" In a deck of poker cards, at least ( ) cards must be drawn to ensure a \"bomb.\"",
        "answer": "42",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117047,
        "difficulty": null
      },
      {
        "id": "c6cde7fbf65bf9ac19902e3cf5139b07f1f418ca2b4a6307e38460cd5ba97b23",
        "domain": "numina",
        "question": "## 167. Math Puzzle $4 / 79$\n\nThe workers of a company's press shop conducted an initiative shift. 50 tons of press material was the result, exceeding their plan by eight tons. What was the percentage of plan fulfillment?",
        "answer": "119",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181841,
        "difficulty": null
      },
      {
        "id": "68f90c077c3324c717774e002dfe09910f7628d98ff0d8fe4612e88ec2bb1723",
        "domain": "numina",
        "question": "8. Let there be a geometric sequence $\\left(a+\\log _{2} 3\\right),\\left(a+\\log _{4} 3\\right),\\left(a+\\log _{8} 3\\right)$, with a common ratio of $\\qquad$ .",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102739,
        "difficulty": null
      },
      {
        "id": "6aa1f0abbc7d5e829124a57b3ee090e0311cb71eb83cafe9f3ba4c5c05976062",
        "domain": "numina",
        "question": "9-5-1. The numbers 1 and 7 are written on the board. In one move, the greatest common divisor of both numbers on the board is added to each of them. For example, if at some point the numbers 20 and 50 are on the board, they will be replaced by the numbers 30 and 60.\n\nWhat numbers will be on the board after 100 moves? Provide the answers in any order. Answer. 588 and 594.",
        "answer": "588594",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12271,
        "difficulty": null
      },
      {
        "id": "13127f2a56815e1a89c10c81a7625f0e5a0dbdfcbeb69fce267b46073738a37c",
        "domain": "numina",
        "question": "Example 5 Calculate $\\sec \\frac{2 \\pi}{9}+\\sec \\frac{4 \\pi}{9}+\\sec \\frac{6 \\pi}{9}+\\sec \\frac{8 \\pi}{9}$.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136475,
        "difficulty": null
      },
      {
        "id": "f4ca73766e78dcd9a9dd6c0e89152f1d905891bcf51f19add85fa257ce70f0d7",
        "domain": "numina",
        "question": "The alphabet consists of 24 letters, and to describe each permutation of these letters, $24 \\mathrm{~cm}^{2}$ of area is needed. How many times the surface area of the Earth is required to describe all permutations? (The radius of the Earth is $r=6370 \\mathrm{~km}$.)",
        "answer": "2921000",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 175656,
        "difficulty": null
      },
      {
        "id": "eb960cb60062f7e8eb54664c3914704b3105af264ffd97ef3b2b316c6b6ccf71",
        "domain": "numina",
        "question": "Example $\\mathbf{3}$ Given the quadratic equation in $x$, $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$, has real roots, find the value of the real number $a$.\n\nThe equation is $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95677,
        "difficulty": null
      },
      {
        "id": "91b552a69252702c5d864c260982c216bb439c803734cf44ecd05ccc77341c1a",
        "domain": "numina",
        "question": "3. Find the sum of all four-digit numbers in which the digits $0,4,5,9$ do not appear.",
        "answer": "6479352",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 4100,
        "difficulty": null
      },
      {
        "id": "86b6bd9fd642a00e690aaa6da313818d2189f9e8f07332356f7357dff899768b",
        "domain": "numina",
        "question": "9. (6 points) A hollow cone and a hollow cylinder, with base inner radii and heights of $12 \\mathrm{~cm}, 20 \\mathrm{~cm}$ respectively, are combined to form a vertically placed container as shown in the figure. Some fine sand is poured into this container, filling the cone and part of the cylinder. It is measured that the sand in the cylinder is $5 \\mathrm{~cm}$ high. If the container is inverted, the height of the sand will be $\\qquad$ $\\mathrm{cm}$.",
        "answer": "11\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114857,
        "difficulty": null
      },
      {
        "id": "00f95325ac504dae6bf7507d752597ff0acebb34948229cb02a0927f90c439c8",
        "domain": "numina",
        "question": "Example 2 Given the function $f(x)=x^{2}+2 x \\cdot \\tan \\theta-1, x \\in[-1, \\sqrt{3}]$, where $\\theta \\in\\left(-\\frac{\\pi}{2}, \\frac{\\pi}{2}\\right)$\n(1) When $\\theta=-\\frac{\\pi}{6}$, find the maximum and minimum values of the function $f(x)$;\n(2) Find the range of $\\theta$ such that $y=f(x)$ is a monotonic function on $[-1, \\sqrt{3}]$.",
        "answer": "-\\frac{4}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132561,
        "difficulty": null
      },
      {
        "id": "1a1a4781a2a17e86a3dbc593df73ec739484a5982772bf00dddbc94e3dc01d6b",
        "domain": "numina",
        "question": "# 5. Option 1.\n\nThe chocolate bar has the shape of a square with a side of 100 cm, divided into pieces with a side of 1 cm. The sweet tooth ate all the pieces along each of the four sides, as shown in the figure. How many pieces did the sweet tooth eat in total?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7eebe7f078c503a48a3eg-4.jpg?height=279&width=287&top_left_y=419&top_left_x=911)",
        "answer": "784",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17783,
        "difficulty": null
      },
      {
        "id": "3acd6ba4a9fd3252dc9008f1933a4bd456d44753358dcb9f695788ac7ff32710",
        "domain": "numina",
        "question": "12. If $p$ is the smallest positive prime such that for some integer $n, p$ divides $n^{2}+5 n+23$, then $p=$",
        "answer": "13",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118359,
        "difficulty": null
      },
      {
        "id": "d35df4cc4c36bb343d1b9ab7e094c45e9bdcdecf416f8fcb9fc9dacb2be6aa9e",
        "domain": "numina",
        "question": "8. Given positive integers $a_{1}, a_{2}, \\cdots, a_{2018}$ satisfying\n$$\na_{1}<a_{2}<\\cdots<a_{2018} \\text {. }\n$$\n\nFor $i=1,2, \\cdots, 2018, b_{i}$ is the number of positive integers in $a_{1}, a_{2}, \\cdots, a_{2018}$ that do not exceed $i$. Then\n$$\n\\frac{\\sum_{k=1}^{2018} a_{k}+\\sum_{k=1}^{a_{2018}} b_{k}}{a_{2018}+1}=\n$$\n$\\qquad$",
        "answer": "2018",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100642,
        "difficulty": null
      },
      {
        "id": "f804a9e59d15b271f2fe790a15d35f5949e558f4db3f3c3a89ade9aca34c5534",
        "domain": "numina",
        "question": "We draw the midlines on the base of a regular tetrahedron. We connect the midpoint of each midline with the vertices of the side face parallel to the midline. What fraction of the volume of the tetrahedron is the volume of the common part of the three tetrahedra thus obtained?",
        "answer": "\\frac{1}{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170408,
        "difficulty": null
      },
      {
        "id": "657d70a5aeffecd47b456e0acf3bec03088c7294ec97124de9be3ac203818f4c",
        "domain": "numina",
        "question": "12. If $16^{\\sin ^{2} x}+16^{\\cos ^{2} x}=10$, then $\\cos 4 x=$\n$\\qquad$ .",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124862,
        "difficulty": null
      },
      {
        "id": "3d4d1a87c492951be56aac7c385ab5830523bc56061dcd1ce282321b9b99089f",
        "domain": "numina",
        "question": "Example 9. The probability of manufacturing a non-standard part is $p=0.004$. Find the probability that among 1000 parts there will be 5 non-standard ones.",
        "answer": "0.1428",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 33074,
        "difficulty": null
      },
      {
        "id": "cb69d9195d8556dcbb2f6d213dfd59847bc2a628f004f4ca3c3d1e8899c60ca6",
        "domain": "numina",
        "question": "3. (8 points) Xiaopang used all of this month's salary to buy a stock. On the first day, the stock price increased by $\\frac{1}{10}$, on the second day it decreased by $\\frac{1}{11}$, on the third day it increased by $\\frac{1}{12}$, and on the fourth day it decreased by $\\frac{1}{13}$. At this point, the value of his stock is exactly 5000 yuan. What was Xiaopang's salary this month? $\\qquad$ yuan.",
        "answer": "5000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118887,
        "difficulty": null
      },
      {
        "id": "6ae6549ec8fd985c8903db9f8222ff3637981bbd7d12bac9558aaf2c8e589340",
        "domain": "numina",
        "question": "2. Arrange the order of operations in the expression\n\n$$\n1891-(1600: a+8040: a) \\times c .\n$$\n\nand calculate its value when $a=40$ and $c=4$. Show how the expression can be modified without changing its numerical value.",
        "answer": "927",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42763,
        "difficulty": null
      },
      {
        "id": "695f8b7533802113bf051ebb0eaec9cd6520b93b4541b6bdae9f342fb9556f28",
        "domain": "numina",
        "question": "21.22 * Write an $n$-digit number using the digits 1 and 2, where no two adjacent digits are both 1, and let the number of such $n$-digit numbers be $f(n)$. Find $f(10)$.",
        "answer": "144",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 140342,
        "difficulty": null
      },
      {
        "id": "6ceadcf7bc106e0e7ca1b6056b1e0a2ec981e5a856afd89199ba9f48172fb22a",
        "domain": "numina",
        "question": "Task 1. (3 points) A piece of cheese is given. It is allowed to choose any positive real number $a$, different from 1, and to cut the piece in the ratio 1 : a by weight, then to cut one of the resulting pieces in the same ratio, and so on. Is it possible with a finite number of cuts to obtain pieces of cheese that can be distributed into two portions of the same weight?",
        "answer": "1+\\sqrt{2}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 177101,
        "difficulty": null
      },
      {
        "id": "7eaa032241e559827ad27503d21353f52283570daafa55d2e5d469e1faf2190f",
        "domain": "numina",
        "question": "The sum of three numbers is 100, two are prime and one is the sum of the other two.\n\n(a) What is the largest of the three numbers?\n(b) Give an example of such three numbers.\n\n(c) How many solutions exist for this problem?",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74948,
        "difficulty": null
      },
      {
        "id": "33ea785d2a3794504c8d7bf1cc739d8c9bd803538da363fe1d594a7a367fcd21",
        "domain": "numina",
        "question": "B2. How many natural numbers less than 100000 have a sum of digits divisible by 9 or 5?",
        "answer": "31109",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 163098,
        "difficulty": null
      },
      {
        "id": "17d309e28446af1af9a51c560b632142de10cca218593dc2ee4d34967ccbcf3e",
        "domain": "numina",
        "question": "5. (3 points) Three-digit even numbers $A, B, C, D, E$ satisfy $A<B<C<D<E$. If $A+B+C+D+E=4306$, then $A$ is the smallest. $\\qquad$",
        "answer": "326",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144369,
        "difficulty": null
      },
      {
        "id": "799c314f5edef39938d24e5c2bbfece2a3786cde1f7e5f0a5f42a700027123a4",
        "domain": "numina",
        "question": "(12) The sum of all real roots of the equation $\\left|x^{2}-3 x+2\\right|+\\left|x^{2}+2 x-3\\right|=11$ is $\\qquad$ .",
        "answer": "\\frac{-19+5\\sqrt{97}}{20}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130868,
        "difficulty": null
      },
      {
        "id": "8dc2468e1413c41845c89c91a82480470ea0c08245f49f94981d1c3787c48c6e",
        "domain": "numina",
        "question": "11. From the sequence of positive integers $1,2,3,4,5, \\cdots$, remove the multiples of 3 and 4, but retain all multiples of 5 (for example, 15 and 120 are retained). After removing, the remaining numbers form a new sequence: $1,2,5,7,10, \\cdots$. Find the 2010th term of the new sequence.",
        "answer": "3349",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 126222,
        "difficulty": null
      },
      {
        "id": "93f38b42f19ba7a78790c933d487d2ce40cfa98811fd66fa9b67fd36a7e598fb",
        "domain": "numina",
        "question": "Example 1 Determine the periodicity of the function $f(x)=x-[x], x \\in \\mathbf{R}$ (where $[x]$ denotes the greatest integer not exceeding $x$) and sketch its graph.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118914,
        "difficulty": null
      },
      {
        "id": "a2551b318f26035e1e32929ddbbf96edcdd0b3690ddb66e7b4f28c3b3427e145",
        "domain": "numina",
        "question": "7. Find the smallest positive integer that is $20 \\%$ larger than one integer and $19 \\%$ smaller than another.",
        "answer": "162",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154674,
        "difficulty": null
      },
      {
        "id": "2ab5db83bd6fe77b01df05e05011f30e934fba3664a3f0afe64d04e09a5800a2",
        "domain": "numina",
        "question": "Many years ago, a teacher who did not want to give a lecture ordered his students to calculate the sum of the numbers from 1 to 100. A very clever student named Gauss found a very simple way to accomplish the task by discovering the formula:\n\n$$\n1+2+3+\\ldots+n=\\frac{n(n+1)}{2}\n$$\n\nSince this story is from a long time ago, today's challenges given to students require more elaborate tasks.\n\n(a) Verify that every prime number greater than 3 leaves a remainder of 1 or 5 when divided by 6.\n\n(b) Verify that the sum of the prime numbers that are greater than 1 and less than 1000 is less than 166338.\n\n(c) In the above estimate, to avoid technical complications, we did not eliminate some numbers that are certainly not prime. Eliminate some of these numbers and verify that the sum of the primes that are greater than 1 and less than 1000 is less than 166000.",
        "answer": "166000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74561,
        "difficulty": null
      },
      {
        "id": "05dedda5584486211d80c9750d34b226f885ad0b8195bac1ba29af805fd1e20d",
        "domain": "numina",
        "question": "The point $K$ lies in the interior of the unit circle. Four lines are drawn through $K$ such that each two adjacent lines form an angle of $45^{\\circ}$, as shown in the picture. In such a way the circle is divided into 8 regions. Four of these regions are colored such that no two colored regions share more than one point. Find the maximal and minimal possible value for the total area of colored regions.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2f0af4b96a9a083248abg-2.jpg?height=414&width=417&top_left_y=381&top_left_x=842)\n\n#",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 90468,
        "difficulty": null
      },
      {
        "id": "2aa2b693d1ae801dc5ad08954c8393e64261db1fcbd11a3749a49d843aa5c3c9",
        "domain": "numina",
        "question": "[ Inequality problems. Case analysis]\n\nLёsha and Ira live in a house where each floor has 9 apartments (the house has one entrance). The floor number of Lёsha is equal to the apartment number of Ira, and the sum of their apartment numbers is 329. What is the apartment number of Lёsha?",
        "answer": "296",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 68208,
        "difficulty": null
      },
      {
        "id": "b36e04fab3b6017dc50a90f7f1b9eb402cb4d7d40a0b3076ef6190e594f12998",
        "domain": "numina",
        "question": "6. On the faces $BCD, ACD, ABD$, and $ABC$ of the tetrahedron $ABCD$, points $A_{1}, B_{1}, C_{1}$, and $D_{1}$ are marked, respectively. It is known that the lines $AA_{1}, BB_{1}, CC_{1}$, and $DD_{1}$ intersect at point $P$, and $\\frac{AP}{A_{1}P}=\\frac{BP}{B_{1}P}=\\frac{CP}{C_{1}P}=\\frac{DP}{D_{1}P}=r$. Find all possible values of $r$.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14982,
        "difficulty": null
      },
      {
        "id": "c781fcfaf8ca0fae2f04f45c4d04a9db4d4f97bfcce6625ceffed998c2038850",
        "domain": "numina",
        "question": "My only son was born when I was 37 years old. This was exactly 32 years after my grandfather's death, and he died at the age of 64. My grandfather was 12 years older than my grandmother, and they got married in 1947, when my grandmother was 18 years old.\n\nIn which year was my son born?\n\nNote: Ignore any discrepancies related to specific birth dates; you can assume, for example, that all the people mentioned have their birthdays on the same day.\n\n(M. Smitková)\n\nHint: How old was the grandfather when he got married?",
        "answer": "2013",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76791,
        "difficulty": null
      },
      {
        "id": "9cb3abc8349225da2b5ec6fa6800c132ef535dbfb0d6bfcbd28ee8ef262e7a2d",
        "domain": "numina",
        "question": "6. Given that a line with slope $k$ is drawn through a focus of the ellipse $x^{2}+2 y^{2}=3$, intersecting the ellipse at points $A$ and $B$. If $AB=2$, then $|k|=$ $\\qquad$ .",
        "answer": "\\sqrt{1+\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122636,
        "difficulty": null
      },
      {
        "id": "a1f077dbc62e6f90a53385dd84492c0fd2795d27aa7ecee893015f32817d93dd",
        "domain": "numina",
        "question": "[  [Theorem on the lengths of a tangent and a secant; the product of the entire secant and its external part. Pythagorean Theorem (direct and inverse). Area of a trapezoid\n\nA circle passes through vertices $A$ and $B$ of rectangle $A B C D$ and is tangent to side $C D$ at its midpoint. A line is drawn through vertex $D$ that is tangent to the same circle at point $E$, and then intersects the extension of side $A B$ at point $K$.\n\nFind the area of trapezoid $B C D K$, if $A B=10$ and $K E: K A=3: 2$.\n\n#",
        "answer": "210",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67497,
        "difficulty": null
      },
      {
        "id": "c3146cd32a95a06e13e95120467322bd2d3f34c0b991acf94d4158e919566383",
        "domain": "numina",
        "question": "9. In the Pokémon factory, there is a type of fruit can, the original packaging bottle's internal is a cylinder with a diameter of 18 cm and a height of 10 cm. Now the internal diameter of the packaging bottle is reduced by 6 cm, and the height is increased by 2 cm. Then, the capacity of the original 1000 bottles is equivalent to the capacity of $\\qquad$ bottles now.",
        "answer": "1875",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126182,
        "difficulty": null
      },
      {
        "id": "df161f6f7f60d6ae8c1f747ad159540698108a1022c9bc69e59b9fc0a343a856",
        "domain": "numina",
        "question": "5. A square and a regular hexagon are drawn with the same side length. If the area of the square is $\\sqrt{3}$, what is the area of the hexagon?",
        "answer": "\\frac{9}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152133,
        "difficulty": null
      },
      {
        "id": "01bb7ebe22376af72044611a321addb92d025dbd51f9e9af51d765c18b66966a",
        "domain": "numina",
        "question": "2. Circle $k$ is inscribed in trapezoid $A B C D, A B \\| C D$, touching side $A B$ at point $E$. If $A E=15, B E=10$ and $C D=8$, determine the radius of circle $k$.",
        "answer": "4\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 161158,
        "difficulty": null
      },
      {
        "id": "e17e4e1fb6a0b63ac1dfe76c3ac53bd7c5f2e2622786ed652ae8dddf9396062d",
        "domain": "numina",
        "question": "## Task 5 - 030725\n\nIn a box, there are 70 balls, namely 20 red, 20 green, 20 yellow, and the rest are black or white. Brigitte is supposed to take out as many balls from this box in the dark so that among them, there are certainly at least 10 balls of the same color.\n\nHow many balls does she have to take out at least? Justify your answer!",
        "answer": "38",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183503,
        "difficulty": null
      },
      {
        "id": "856d16e550fc596b76f36972826bb2d37a96f93df277bcc40625c404d2688d36",
        "domain": "numina",
        "question": "6. (10 points) Fill in the 6 boxes in the figure with the numbers $1 \\sim 6$ respectively, the smallest result that can be obtained is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "342",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 104801,
        "difficulty": null
      },
      {
        "id": "27f42e9ee2b90fb7b92621fc9a6be6c47942dc083ca97f9084a8eeb520a865f5",
        "domain": "numina",
        "question": "20. As shown in the figure, the shape is composed of ten line segments. Now, we are to color the line segments using three colors, with the requirement that adjacent segments must be colored differently (line segments sharing a common endpoint are considered adjacent). If colors can be reused, how many different coloring methods are there in total?\n\nThe shape is composed of ten line segments. Now, we are to color the line segments using three colors, with the requirement that adjacent segments must be colored differently (line segments sharing a common endpoint are considered adjacent). If colors can be reused, how many different coloring methods are there in total?",
        "answer": "30",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 94551,
        "difficulty": null
      },
      {
        "id": "ab536c5546bf00bcc403e4626ec7488e346895faf8be4e98064f276c3d73d03a",
        "domain": "numina",
        "question": "3. In triangle $\\triangle A B C$, the base $A B$ of which lies on the x-axis, altitudes $A M, B N$ and $C K$ are drawn. Find the length of the base $A B$, if the coordinates of points $M(2,2)$ and $N(4,4)$ are known.",
        "answer": "4\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6096,
        "difficulty": null
      },
      {
        "id": "9800f27445931da4335eabe78c186f9bc5076250d3238291f4e518ce033b2707",
        "domain": "numina",
        "question": "10. Let $p$, $q$, $r$ be prime numbers, and\n$$\np|(q r-1), q|(r p-1), r \\mid(p q-1) \\text {. }\n$$\n\nFind all possible values of $p q r$.",
        "answer": "30",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 133612,
        "difficulty": null
      },
      {
        "id": "b834766628ea99b896cad0200f1429006eda0938f64ef7a7cccb39e1fbd3e5e0",
        "domain": "numina",
        "question": "4. Let $a, b \\in \\mathbf{N}^{*},(a, b)=1$. Find the number of natural numbers $c$ such that the indeterminate equation $a x+b y=c$ has no non-negative integer solutions.",
        "answer": "\\frac{1}{2}(1)(1)",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 91601,
        "difficulty": null
      },
      {
        "id": "dd44c3ccf674e3e0e18116fcbba80ff2d4536c59c4f486eac90fc40a5c08d83f",
        "domain": "numina",
        "question": "495. Find all pairs of natural numbers if their sum is 60, and the least common multiple is 72.",
        "answer": "2436",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 45192,
        "difficulty": null
      },
      {
        "id": "054c7f8e60013230bcd5a1448f8c6e21b3cddf5fef5e4e1654b341863994e580",
        "domain": "numina",
        "question": "7. Given $z \\in \\mathbf{C}$. If the equation $x^{2}-2 z x+\\frac{3}{4}+\\mathrm{i}=0$ (where $\\mathrm{i}$ is the imaginary unit) has real roots, then the minimum value of $|z|$ is . $\\qquad$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131832,
        "difficulty": null
      },
      {
        "id": "26b1fb4c4ad9b82818c4d67d74cdde0d8b8e753f957e526dcf21d3607c66234e",
        "domain": "numina",
        "question": "Evdokimov M.A.\n\nIn each cell of a $5 \\times 5$ board, there is either a cross or a zero, and no three crosses stand in a row either horizontally, vertically, or diagonally. What is the maximum number of crosses that can be on the board?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 52850,
        "difficulty": null
      },
      {
        "id": "3730247e1cfd2add0e5c3630cd8d3c63ac125052f627e444a2061d4ffa7a9a92",
        "domain": "numina",
        "question": "4. If $\\cos ^{4} \\theta+\\sin ^{4} \\theta+(\\cos \\theta \\cdot \\sin \\theta)^{4}+$ $\\frac{1}{\\cos ^{4} \\theta+\\sin ^{4} \\theta}=\\frac{41}{16}$, then $\\sin ^{2} \\theta=$",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98842,
        "difficulty": null
      },
      {
        "id": "a6f9f31bb02002b5522d6a28fc40e7cafe42d2e98985f8fee0cf8aad85671a2e",
        "domain": "numina",
        "question": "Let $x_{1}=1$ and $x_{n+1}=x_{n}+\\left\\lfloor\\frac{x_{n}}{n}\\right\\rfloor+2$ for $n=1,2,3, \\ldots$, where $\\lfloor x\\rfloor$ denotes the largest integer not greater than $x$. Determine $x_{1997}$.",
        "answer": "23913",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 72744,
        "difficulty": null
      },
      {
        "id": "3fcc9b5768e9e61b22ac685ad7e5cee3dff9f8001a1b10f79c02f53a4445ceef",
        "domain": "numina",
        "question": "4. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=$ $2 \\sqrt{2}, B C=A A_{1}=2, P$ is a moving point on the diagonal $A C_{1}$, $Q$ is a moving point on the base $A B C D$ (points $P$ and $Q$ can coincide). Then the minimum value of $B_{1} P+P Q$ is $\\qquad$",
        "answer": "4.3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 95491,
        "difficulty": null
      },
      {
        "id": "8adb2e412ee35b01f6d4c5cd951acd08509cfe285cd06ad25717da850005c30b",
        "domain": "numina",
        "question": "A cube with an edge of $12 \\mathrm{~cm}$ was divided into smaller, identical cubes such that the sum of the surface areas of all the smaller cubes was eight times greater than the surface area of the original cube.\n\nDetermine how many small cubes there were and what the length of their edges was.\n\n(M. Volfová)",
        "answer": "512",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76882,
        "difficulty": null
      },
      {
        "id": "30019a10c9aa5cafb6832a433e47164e365c37d57c8a11c4521621cc32a2faa5",
        "domain": "numina",
        "question": "13. On a square $A B C D$ a line segment $B E$ is drawn such that the point $E$ lies on the side $C D$. The perimeter of triangle $B C E$ is three-quarters of the perimeter of the square $A B C D$. The ratio of lengths $C E: C D$ is $\\lambda: 1$.\nWhat is the value of $960 \\times \\lambda$ ?",
        "answer": "720",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156363,
        "difficulty": null
      },
      {
        "id": "09d5252a06cdd08cd6b6d9dd58d5e3cce7908fa52c1fcfa8f84adf0e93ce4c06",
        "domain": "numina",
        "question": "In the diagram, $\\triangle A B C$ is right-angled at $C$. Point $D$ is on $A C$ so that $\\angle A B C=2 \\angle D B C$. If $D C=1$ and $B D=3$, determine the length of $A D$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_fa78072b1e75930324fdg-3.jpg?height=328&width=371&top_left_y=1170&top_left_x=931)",
        "answer": "\\frac{9}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194423,
        "difficulty": null
      },
      {
        "id": "af48bc224627a195c5d206da55aee2a32f10fde74954de2f31ba93b6b6e25780",
        "domain": "numina",
        "question": "Mrs. Kucera was on a seven-day vacation, and Káta walked her dog and fed her rabbits for the entire duration. For this, she received a large cake and 700 CZK. After another vacation, this time a four-day one, Káta received the same cake and 340 CZK for walking and feeding according to the same rules.\n\nWhat was the value of the cake?",
        "answer": "140",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76434,
        "difficulty": null
      },
      {
        "id": "480f10547bf3d58e5d03dca3275552de398de1310520e0e22e002e6198d7be85",
        "domain": "numina",
        "question": "3rd Swedish 1963 Problem 2 The squares of a chessboard have side 4. What is the circumference of the largest circle that can be drawn entirely on the black squares of the board?",
        "answer": "2\\sqrt{40}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150112,
        "difficulty": null
      },
      {
        "id": "dbe13de7a06110230916e369ab3b52b9a557f993ce3c75ac49df64c5a713a555",
        "domain": "numina",
        "question": "3. Find the value of the expression $\\operatorname{tg} 20^{\\circ}+4 \\sin 20^{\\circ}$.",
        "answer": "\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3729,
        "difficulty": null
      },
      {
        "id": "ca690058e272d96a2b6bc6cd7d1f9fb2cca97ff4e1c919ac483463a700e87f0f",
        "domain": "numina",
        "question": "3. Let $O$ be the circumcenter of acute $\\triangle A B C$, with $A B=6, A C=10$. If $\\overrightarrow{A O}=x \\overrightarrow{A B}+y \\overrightarrow{A C}$, and $2 x+10 y=5$, then $\\cos \\angle B A C=$ $\\qquad$ .",
        "answer": "\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 139039,
        "difficulty": null
      },
      {
        "id": "c48d1093c93c6dbe2982763a0e1b3e3bcb2240bde8b28ad910c677908befdc5c",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the volumes of the bodies bounded by the surfaces.\n\n$$\n\\frac{x^{2}}{16}+\\frac{y^{2}}{9}+\\frac{z^{2}}{36}=1, z=3, z=0\n$$",
        "answer": "11\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47242,
        "difficulty": null
      },
      {
        "id": "e6ab50dccb4278b41056f7e834b50a604bfe3b3401a390238934b6c40bd6076f",
        "domain": "numina",
        "question": "2. (7 points) In rectangle $A B C D$, side $A B$ is equal to 6, and side $B C$ is equal to 11. Bisectors of the angles from vertices $B$ and $C$ intersect side $A D$ at points $X$ and $Y$ respectively. Find the length of segment $X Y$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2509cbb2376df96befe7g-1.jpg?height=317&width=417&top_left_y=1121&top_left_x=822)\n\n## Answer: 1.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14824,
        "difficulty": null
      },
      {
        "id": "0751cdf1b7d7fbfe617be9b512ee211d7b2f9a20b2c147a100b01af91f322a22",
        "domain": "numina",
        "question": "The base of the right prism $A B C A_{1} B_{1} C_{1}$ is an isosceles triangle $A B C$, where $A B=B C=5$, $\\angle A B C=2 \\arcsin 3 / 5$. A plane perpendicular to the line $A_{1} C$ intersects the edges $A C$ and $A_{1} C_{1}$ at points $D$ and $E$ respectively, such that $A D=1 / 3 A C, E C_{1}=1 / 3 A_{1} C_{1}$. Find the area of the section of the prism by this plane.",
        "answer": "\\frac{40}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66341,
        "difficulty": null
      },
      {
        "id": "eab9c45fec3007de9bb1feddbefba1bd667db8d8fc404d204582bf99fff61162",
        "domain": "numina",
        "question": "3.1. ** Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence with common difference $d, d \\neq 0$, and $\\left\\{b_{n}\\right\\}$ be a geometric sequence with common ratio $q$. Given that $a_{1}=b_{1}, a_{3}=b_{3}, a_{7}-b_{5}$. Find the relationship between $n$ and $m$ when $a_{n}=b_{m}$.",
        "answer": "2^{(+1)/2}-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111541,
        "difficulty": null
      },
      {
        "id": "bda7c8e65dc8f23fb8f38a95a9a4e17c4c1c042b6a59fc0770450c5c19ee0a46",
        "domain": "numina",
        "question": "6 Set $A=\\{0,1,2,3,4,5,9\\}, a, b \\in A$, and $a \\neq b$. The number of functions of the form $y=-a x^{2}+(4-b) x$ whose graph vertex is in the first quadrant is $\\qquad$ .",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132129,
        "difficulty": null
      },
      {
        "id": "f755b0699a874eaa931d2578d5a7483ac24c57311f4ad3b03c082168adf8959b",
        "domain": "numina",
        "question": "IMO 1995 Problem A3 Determine all integers n > 3 for which there exist n points A 1 , ... , A n in the plane, no three collinear, and real numbers r 1 , ... , r n such that for any distinct i, j, k, the area of the triangle A i A j A k is r i + r j + r k .",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 149758,
        "difficulty": null
      },
      {
        "id": "2af10f3f754d8bad98cdffa2905c8faadd1de8d45325bb7395ba09157d7b4aa1",
        "domain": "numina",
        "question": "$\\left[\\begin{array}{l}\\text { Tangent Circles } \\\\ \\text { [Pythagorean Theorem (direct and inverse).] }\\end{array}\\right]$\n\nTwo circles with radii $\\sqrt{19}$ and $\\sqrt{76}$, touching each other externally, are inscribed in a semicircle (i.e., each circle touches this semicircle and its diameter). Find the radius of the semicircle.",
        "answer": "4\\sqrt{19}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65909,
        "difficulty": null
      },
      {
        "id": "b4036b077d9b1d9a0f133d1ebf93937dc769c25f7f6eb60ff2b685bf6a751cac",
        "domain": "numina",
        "question": "4. In the plane, there is a set $S$ of $n$ points ( $n>2$ ) with the property: if $A, B \\in S$, then there exists $C \\in S$ such that triangle $A B C$ is equilateral. How many can the number $n$ be?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189658,
        "difficulty": null
      },
      {
        "id": "792b9b413d3d972f382e72eeac43dd95f3a2b7a0cec00377128dc0cffa961cf4",
        "domain": "numina",
        "question": "6. (10 points) Expression: $\\frac{\\frac{1}{1}-\\frac{1}{3}}{\\frac{1}{1} \\times \\frac{1}{2} \\times \\frac{1}{3}}+\\frac{\\frac{1}{2}-\\frac{1}{4}}{\\frac{1}{2} \\times \\frac{1}{3} \\times \\frac{1}{4}}+\\frac{\\frac{1}{3}-\\frac{1}{5}}{\\frac{1}{3} \\times \\frac{1}{4} \\times \\frac{1}{5}}+\\cdots+\\frac{\\frac{1}{9}-\\frac{1}{11}}{\\frac{1}{9} \\times \\frac{1}{10} \\times \\frac{1}{11}}$ calculation result",
        "answer": "108",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128979,
        "difficulty": null
      },
      {
        "id": "95fc84f66919602174ca8785546cb87687e32dc89029457d9bf13877994018bc",
        "domain": "numina",
        "question": "A car with a weight of $P=1800 \\mathrm{~kg}$ starts on a road where $\\rho=\\frac{1}{50}, e=\\frac{1}{40} ; 1 \\frac{1}{12}$ minutes after, while covering $900 \\mathrm{~m}$, it reaches its maximum speed: $v=12 \\frac{\\mathrm{m}}{\\mathrm{sec}}$. How many horsepower is its power? What is the traction force?[^0]\n\n\n[^0]:    ${ }^{1}$ In this problem, $\\rho$ is the coefficient of friction, and $e$ is the sine of the incline angle.",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 174898,
        "difficulty": null
      },
      {
        "id": "504e890dc55a349e9eecccb518e711b983af415ea4d70088a59085f6cf92aeac",
        "domain": "numina",
        "question": "(12) The complex number $z$ satisfies $|z|(3 z+2 \\mathrm{i})=2(\\mathrm{i} z-6)$, then $|z|=$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132711,
        "difficulty": null
      },
      {
        "id": "270c2373ca71682cad4d1de83b66b9f260d1f443b6b04a208b333a7eac7580c8",
        "domain": "numina",
        "question": "4. The sequence $\\left\\{x_{n}\\right\\}$ is defined as follows: $x_{1}=\\frac{1}{2}, x_{k+1}=x_{k}^{2}+x_{k}, k=1,2, \\cdots$. Find the integer part of the following sum:\n$$\n\\frac{1}{1+x_{1}}+\\frac{1}{1+x_{2}}+\\cdots+\\frac{1}{1+x_{2005}}\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 121948,
        "difficulty": null
      },
      {
        "id": "d8334b992efe10b53d9294c336672e63729dc8e6c4afd5e7103a3110e322d3c6",
        "domain": "numina",
        "question": "4. for $a<0.5$\n\n## Correct answer: for $a<1$\n\n## Question 3\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\n\nQuestion 4\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\n\n## Question 5\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\nWorker $A$ and worker $B$ can complete the work in 7.2 days, worker $A$ and worker $C$ in 9 days, worker $B$ and worker $C$ in 12 days. How many days will they take to complete the work if they all work together?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4658,
        "difficulty": null
      },
      {
        "id": "0c7098bf6c074da2c931550657f39e1481ae86f13e52f80fb9d04ea6b184e043",
        "domain": "numina",
        "question": "Problem 6.7. Anya places pebbles on the sand. First, she placed one stone, then added pebbles to form a pentagon, then made a larger outer pentagon with pebbles, then another outer pentagon, and so on, as shown in the picture. The number of stones she had arranged on the first four pictures: 1, 5, 12, and 22. If she continues to create such pictures, how many stones will be on the 10th picture?\n\n- \n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=113&width=104&top_left_y=818&top_left_x=382)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=187&width=186&top_left_y=823&top_left_x=527)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=266&width=277&top_left_y=820&top_left_x=758)",
        "answer": "145",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15961,
        "difficulty": null
      },
      {
        "id": "5603dc1a3a1157dabb1830d27d2093ecb22e79680a46a5962a5ca8e2cd97cf74",
        "domain": "numina",
        "question": "B4. For the second round of the Math Olympiad, 999 students are invited. Melanie prepares invitation letters in the order of participant numbers: $1,2,3, \\ldots$ For some values of $n \\geqslant 100$, she notices the following: the number of participant numbers from $1$ to $n$ that end in a 5 is exactly equal to the number formed by the last two digits of $n$.\n\nFor how many values of $n$ (with $100 \\leqslant n<1000$) does this hold?",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85546,
        "difficulty": null
      },
      {
        "id": "9a08bbd6cab5ea4b18ea832d3a73d7c13426d17956fa0da26573a7137ae6cc87",
        "domain": "numina",
        "question": "The family approached the bridge at night. Dad can cross it in 1 minute, Mom in 2, the child in 5, and Grandma in 10 minutes. They have one flashlight. The bridge can only hold two people at a time. How can they cross the bridge in 17 minutes? (If two people cross together, they move at the slower of their speeds. Moving across the bridge without a flashlight is not allowed. Shining the light from a distance is not allowed. Carrying each other is not allowed.)",
        "answer": "17",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56807,
        "difficulty": null
      },
      {
        "id": "436c57ce5d89aedd2e4c51ad522442abac313464e08a520eda2849d72efd4b53",
        "domain": "numina",
        "question": "【16】There are four points $A, B, C, D$ arranged from left to right on a straight line. Using these four points as endpoints, 6 line segments can be formed. It is known that the lengths of these 6 line segments are $14, 21, 34, 35, 48, 69$ (unit: millimeters). Then the length of line segment $\\mathrm{BC}$ is ( ) millimeters.",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 111066,
        "difficulty": null
      },
      {
        "id": "f45a74967303c5d2f23315fb78e52b965a7edc0f290a8aa2e3896a63837ea563",
        "domain": "numina",
        "question": "4B. Around are placed 10 children. Ivo needs to join them. They will all play the elimination game in the following way: they start counting from the number 1 in the direction of the clock's hands, and the one on whom the number 13 falls is eliminated from the game. The counting continues with the remaining children where it was interrupted, and the thirteenth one is again eliminated from the game, etc. At which position (between which two of the ten marked numbers, see image) should Ivo stand to be the last one in the game?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_a6c2bf676fcc39861820g-03.jpg?height=377&width=360&top_left_y=840&top_left_x=666)",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 190205,
        "difficulty": null
      },
      {
        "id": "4648b2b5088b05cc7ea61a94b612b1383b63f83ce959f320a9c722f718bf75c7",
        "domain": "numina",
        "question": "$A M$ - bisector of triangle $A B C, B M=2, C M=3, D$ - the point of intersection of the extension of $A M$ with the circumcircle of the given triangle, $M D=2$. Find $A B$.",
        "answer": "\\sqrt{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67237,
        "difficulty": null
      },
      {
        "id": "6fe573b3af20f5b9e90f3f26c550073eb6b6fe6b68eee38f0e562c2fb45c8ebb",
        "domain": "numina",
        "question": "## Task B-1.6.\n\nHow many ordered pairs of natural numbers $(m, n)$ are there for which the number $3^{m}+7^{n}$ is divisible by 10, given that $1 \\leqslant m \\leqslant 80, 81 \\leqslant n \\leqslant 185$?",
        "answer": "2100",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87603,
        "difficulty": null
      },
      {
        "id": "0447accab0c6150e7b9f4b49eb8c18faffe3ebcbb0a70ff3415a755f7d8acc33",
        "domain": "numina",
        "question": "## Task 36/80\n\nThe smallest natural number $n$ is sought for which the following holds\n\n$$\nn \\equiv 1(\\bmod 2) ; \\quad n \\equiv 1(\\bmod 3) ; \\quad n \\equiv 1(\\bmod 5) ; \\quad n \\equiv 0(\\bmod 7) ; \\quad n \\equiv 1(\\bmod 11)\n$$",
        "answer": "1981",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182627,
        "difficulty": null
      },
      {
        "id": "0f62d5022c52c7a95bff8e1c6341ec456dbd7d50f2c120bfc593fe971ac935fe",
        "domain": "numina",
        "question": "I1.2 Given that $f(x)=-x^{2}+10 x+9$, and $2 \\leq x \\leq \\frac{a}{9}$. If $b$ is the difference of the maximum and minimum values of $f$, find the value of $b$.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153232,
        "difficulty": null
      },
      {
        "id": "83f23fbd4451510904586a30e924ab2c733c8d4ddce0e3e02b1614a64e664842",
        "domain": "numina",
        "question": "19. Let $f(x)$ be a function that satisfies the following conditions:\n(1) If $x>y$ and $f(x)+x \\geqslant w \\geqslant f(y)+y$, then there exists a real number $z \\in[y, x]$ such that $f(z)=w-z$;\n(2) The equation $f(x)=0$ has at least one solution, and among these solutions, there is one that is not greater than all the others;\n(3) $f(0)=1$;\n(4) $f(-2007) \\leqslant 2008$;\n(5) $f(x) f(y)=f[x f(y)+y f(x)+x y]$.\n\nFind: the value of $f(-2007)$.",
        "answer": "2008",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102347,
        "difficulty": null
      },
      {
        "id": "15dbda23a2cf8b1ac20254af04ddc785b2e5d234f0f6046bd956d866c3396cfa",
        "domain": "numina",
        "question": "4. Given that $f(x), g(x)$ are functions defined on $\\mathbf{R}$, the graph of $f(x)$ is symmetric about the line $x=1$, the graph of $g(x)$ is symmetric about the point $(1,-2)$, and $f(x)+g(x)=9^{x}+x^{3}+1$, then the value of $f(2) g(2)$ is $\\qquad$ .",
        "answer": "2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128255,
        "difficulty": null
      },
      {
        "id": "34c7c5fef277649c08da68fe798c35f5e32460ee2175a1815c10a4389a6cf9fd",
        "domain": "numina",
        "question": "14th Swedish 1974 Problem 3 Let a 1 = 1, a 2 = 2 a 1 , a 3 = 3 a 2 , a 4 = 4 a 3 , ... , a 9 = 9 a 8 . Find the last two digits of a 9 .",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 148835,
        "difficulty": null
      },
      {
        "id": "418e3a37c9a6d28ff2d3ceaa7ed7e3566cfe3beafc55b485052f06e6265fc9dd",
        "domain": "numina",
        "question": "9. As shown in the figure, in the cube $A B C D-A_{1} B_{1} C_{1} D_{1}$ with edge length 2, $E$ is the midpoint of $B C$, and point $P$ moves along the line segment $D_{1} E$. Then, the minimum distance from point $P$ to the line $C C_{1}$ is $\\qquad$.",
        "answer": "\\frac{2\\sqrt{5}}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142612,
        "difficulty": null
      },
      {
        "id": "a1e58f9d61c4028daee3f6947dabcec89fabaa390c0df6d3e2516f63e2ae0465",
        "domain": "numina",
        "question": "5. (10 points) If $\\frac{O}{11}-\\frac{7}{\\square}<\\frac{4}{5}$ holds, then the maximum sum of the non-zero natural numbers that can be filled in “ $O$ ” and “ $\\square$ ” is",
        "answer": "393",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 96312,
        "difficulty": null
      },
      {
        "id": "55071ed359155b1062244b37c99502f0d5276071dbb7309207715eaf8f92c8f4",
        "domain": "numina",
        "question": "1. Let $n$ be a natural number, $a, b$ be positive real numbers, and satisfy $a+b=2$, then the minimum value of $\\frac{1}{1+a^{n}}+\\frac{1}{1+b^{n}}$ is $\\qquad$.",
        "answer": "1",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 116225,
        "difficulty": null
      },
      {
        "id": "ea87c287fef8e95e2a8d35d716a6a1aac878900fe487162e20fcedaac26ae070",
        "domain": "numina",
        "question": "8. (7 points) A bag contains 10 white balls, 5 black balls, and 4 blue balls. All balls are identical except for their color. If $n$ balls are taken out, what is the minimum value of $n$ to ensure that at least 2 balls of each color are obtained? $\\qquad$ .",
        "answer": "17",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 147227,
        "difficulty": null
      },
      {
        "id": "9fa3d8440a8edc7394874b0bd5e9145199789cbc10455d26ac4638b01d22ba6c",
        "domain": "numina",
        "question": "5. (10 points) Among all the factors of the four-digit number Shuangcheng Shuangcheng, 3 are prime numbers, and the other 39 are not prime. Then, \nthe four-digit number Shuangcheng Shuangcheng has $\\qquad$ factors.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 93081,
        "difficulty": null
      },
      {
        "id": "abb194e316fd16766ce2dba7e22aa0a8a703ee2109b0718b64337ed17e468094",
        "domain": "numina",
        "question": "1. Given that the three interior angles $\\angle A, \\angle B, \\angle C$ of $\\triangle \\mathrm{ABC}$ satisfy $2 \\angle A - \\angle B, 3 \\angle B - 2 \\angle C$, and $90^{\\circ} - \\angle A$, the minimum of these is $\\alpha$, then the maximum value of $\\alpha$ is $\\qquad$",
        "answer": "\\frac{2\\pi}{9}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98102,
        "difficulty": null
      },
      {
        "id": "7d8c8963ab25ed480b367c81119140c740997b0bd47c89eafac80b1500e36834",
        "domain": "numina",
        "question": "Let $f: \\mathbb{Z}_{>0} \\rightarrow \\mathbb{R}$ be a function for which it holds that for all $n>1$ there is a prime divisor $p$ of $n$ such that\n\n$$\nf(n)=f\\left(\\frac{n}{p}\\right)-f(p)\n$$\n\nMoreover, it is given that $f\\left(2^{2014}\\right)+f\\left(3^{2015}\\right)+f\\left(5^{2016}\\right)=2013$.\nCalculate $f\\left(2014^{2}\\right)+f\\left(2015^{3}\\right)+f\\left(2016^{5}\\right)$.",
        "answer": "\\frac{49}{3}",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 195294,
        "difficulty": null
      },
      {
        "id": "47991816cd709428a7f2b3666589a8756f9a38aff0f94966c55087e953b0907e",
        "domain": "numina",
        "question": "15. Let $a>0, x \\in[-1,1]$. When the function $y=-x^{2}-a x+b$ has a minimum value of -1 and a maximum value of 1, find the corresponding values of $x$ for which the function attains its minimum and maximum values.",
        "answer": "-2+2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123334,
        "difficulty": null
      },
      {
        "id": "33e9ae36f196a975d7c179704028dc3219cdf874486ce86492cfd0b8e90e3700",
        "domain": "numina",
        "question": "1. We understand a palindrome as a natural number that reads the same forwards and backwards, for example, 16 261. Find the largest four-digit palindrome whose square is also a palindrome.",
        "answer": "2002",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75159,
        "difficulty": null
      },
      {
        "id": "022c56f59f3398a93117a022c5bb206531a037237150a695a19ddcb2d5b7cef7",
        "domain": "numina",
        "question": "## Task A-4.5.\n\nLet $z_{1}$, $z_{2}$, and $z_{3}$ be complex numbers such that $\\left|z_{1}\\right|=\\left|z_{2}\\right|=\\left|z_{3}\\right|=1$ and $4 z_{3}=$ $3\\left(z_{1}+z_{2}\\right)$. What is $\\left|z_{1}-z_{2}\\right|$?",
        "answer": "\\frac{2}{3}\\sqrt{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87654,
        "difficulty": null
      },
      {
        "id": "ccb96691c53c27d0909ab828e03ff5d03aacf028114689a272c6b039c7a30121",
        "domain": "numina",
        "question": "Garkovv S.B.\n\nThe sequence $\\left(a_{n}\\right)$ is such that $a_{n}=n^{2}$ for $1 \\leq n \\leq 5$ and for all natural $n$ the equality $a_{n+5} + a_{n+1} = a_{n+4} + a_{n}$ holds. Find $a_{2015}$.",
        "answer": "17",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70199,
        "difficulty": null
      },
      {
        "id": "0ef26a7ba385127f74866f83b0fd060627d04e0a62e0cd67b60d64c08af7965c",
        "domain": "numina",
        "question": "Shapovalov A.V.\n\nFor what largest $n$ can one choose $n$ points on the surface of a cube so that not all of them lie in the same face of the cube and are vertices of a regular (planar) $n$-gon.",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60733,
        "difficulty": null
      },
      {
        "id": "53be383ddb32b9ec96a65040bfbe98aedcfd19a5925a1ec9b4b26cc4848f547e",
        "domain": "numina",
        "question": "# 9. Problem 9.10\n\nIn a convex 10-gon \\(A_{1} A_{2} \\ldots A_{10}\\), all sides and all diagonals connecting vertices every other one (i.e., \\(A_{1} A_{3}, A_{2} A_{4}\\), etc.) are drawn, except for the side \\(A_{1} A_{10}\\) and the diagonals \\(A_{1} A_{9}, A_{2} A_{10}\\).\n\nWe call a path leading from \\(A_{1}\\) to \\(A_{10}\\) a non-self-intersecting broken line (i.e., one in which non-adjacent segments do not have common points) with endpoints \\(A_{1}\\) and \\(A_{10}\\), each segment of which coincides with one of the drawn sides or diagonals. Find the number of such paths.",
        "answer": "55",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20297,
        "difficulty": null
      },
      {
        "id": "e9c9378ea1691c1deadd2d7ab532b37f5a33a7e859a5a022a4c45400403c9933",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the figure bounded by the graphs of the functions:\n\n$$\ny=(x+1)^{2}, y^{2}=x+1\n$$",
        "answer": "\\frac{1}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47504,
        "difficulty": null
      },
      {
        "id": "7d3cbe34232c1d79d4644671a397092d22ed4fe24377db4ac76b80683e836702",
        "domain": "numina",
        "question": "## Task A-4.2.\n\nA rubber ball was thrown from a height of 200 meters. Each time it bounces off the surface, it reaches $4 / 5$ of the previous height: after the first bounce, it rises to 160 meters, after the second bounce to 128 meters, etc. What is the total distance the ball travels before it comes to a stop?",
        "answer": "1800",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89094,
        "difficulty": null
      },
      {
        "id": "52f87f186a7880f893a82e1d3cec2234d2cc3a352b2834d7bdda43cff35dcbbb",
        "domain": "numina",
        "question": "Problem 8.2. (15 points) Real numbers $x_{1}, x_{2}, x_{3}, x_{4}$ are such that\n\n$$\n\\left\\{\\begin{array}{l}\nx_{1}+x_{2} \\geqslant 12 \\\\\nx_{1}+x_{3} \\geqslant 13 \\\\\nx_{1}+x_{4} \\geqslant 14 \\\\\nx_{3}+x_{4} \\geqslant 22 \\\\\nx_{2}+x_{3} \\geqslant 23 \\\\\nx_{2}+x_{4} \\geq 24\n\\end{array}\\right.\n$$\n\nWhat is the smallest value that the sum $x_{1}+x_{2}+x_{3}+x_{4}$ can take?",
        "answer": "37",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 20001,
        "difficulty": null
      },
      {
        "id": "0df19cbb5f87d2ffda444e790528a45bb3b611958c40348e8de4d97fca44cf27",
        "domain": "numina",
        "question": "6.50 In an exam, there are 4 multiple-choice questions, each with 3 options. A group of students take the exam, and the result is that for any 3 people, there is 1 question where their answers are all different. How many students can participate in the exam at most?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143843,
        "difficulty": null
      },
      {
        "id": "b9afa72153f4bf9393e936b8f56284580d90e1ffec9ec0bb849266c8eee0b4ae",
        "domain": "numina",
        "question": "1. Determine the greatest value of the natural number $n$ such that the number $39^{n}$ is a divisor of the number 39! Explain your answer.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 189387,
        "difficulty": null
      },
      {
        "id": "0276ada275ea64d845bb184e8e1bbbeccc1d0ecd1cfc27ad9201d279589986a9",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{n^{2}-\\sqrt{n^{3}+1}}{\\sqrt[3]{n^{6}+2}-n}$",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46142,
        "difficulty": null
      },
      {
        "id": "b4ae032107fafa30462158fa07f07c60eca57afdc1e95d398d4d253cba7ef8f4",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the equations in polar coordinates.\n\n$$\n\\rho=6(1+\\sin \\varphi),-\\frac{\\pi}{2} \\leq \\varphi \\leq 0\n$$",
        "answer": "12(2-\\sqrt{2})",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47325,
        "difficulty": null
      },
      {
        "id": "939d675327432ffba886721475b1efcfd5e6bc976a8f1a4d05293b910b86c134",
        "domain": "numina",
        "question": "Tokaeva I.\n\nLet $F_{1}, F_{2}, F_{3}, \\ldots$ be a sequence of convex quadrilaterals, where $F_{k+1}$ (for $k=1,2,3, \\ldots$) is obtained by cutting $F_{k}$ along a diagonal, flipping one of the parts, and gluing it back along the cut line to the other part. What is the maximum number of different quadrilaterals that this sequence can contain? (Different polygons are those that cannot be made to coincide by a movement.)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 54721,
        "difficulty": null
      },
      {
        "id": "b051da86877f3a6f74a956d5c9f23ad6bb73c7d9b15ccabf795f096bdee6005b",
        "domain": "numina",
        "question": "1 The minimum value of the function $y=\\sin ^{4} x+\\cos ^{4} x+\\sec ^{4} x+\\csc ^{4} x$ is",
        "answer": "8.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131044,
        "difficulty": null
      },
      {
        "id": "825253583508a795f418319150bf94b621b18f1ca703050d5c3eb101fe14fa66",
        "domain": "numina",
        "question": "Pankrác, Servác, and Bonifác bought a boat. Pankrác paid $60\\%$ of the boat's price, Servác paid $40\\%$ of the remaining price, and Bonifác paid the missing amount, which was 30 gold coins.\n\nHow many gold coins did the boat, which the boys bought, cost?\n\n(L. Hozová)",
        "answer": "125",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76465,
        "difficulty": null
      },
      {
        "id": "bd2d9b537a0c0120a504c199211332955df47484c58d43951dad083be0b88b28",
        "domain": "numina",
        "question": "30.6. Let $A B C$ be an isosceles triangle with $A B=A C$. Let $D$ and $E$ be the feet of the perpendiculars from $B$ and $C$ to $\\overline{A C}$ and $\\overline{A B}$, respectively. Suppose that $\\overline{C E}$ and $\\overline{B D}$ intersect at point $H$. If $E H=1$ and $A D=4$, find $D E$.",
        "answer": "\\frac{8\\sqrt{17}}{17}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 154071,
        "difficulty": null
      },
      {
        "id": "4454727f17df717c97af28400d99c79fc78247087f672506c666bbb7f418243d",
        "domain": "numina",
        "question": "(9) Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy $a_{n}=(2 n-1)(2 n+1)(2 n+3)$, then the greatest common divisor $d$ of $a_{1}, a_{2}, \\cdots$, $a_{2008}$ these 2008 terms is $\\qquad$.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 141077,
        "difficulty": null
      },
      {
        "id": "a662a841aa23b93b452872c059ffef5e26c611327b8a02320b31bca864052504",
        "domain": "numina",
        "question": "18. Numbers $a, b$ and $c$ are such that $\\frac{a}{b+c}=\\frac{b}{c+a}=\\frac{c}{a+b}=k$. How many possible values of $k$ are there?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156408,
        "difficulty": null
      },
      {
        "id": "e9cc004a882fbbb9780876436ca29635a48a6aea1e71f2fe19af45e2c88686ee",
        "domain": "numina",
        "question": "Exercise 3. On a rectangular box with dimensions $6 \\mathrm{~cm} \\times 4 \\mathrm{~cm}$, a ribbon has been placed diagonally as shown in the figure below (the ribbon is represented in gray in the figure).\n\n![](https://cdn.mathpix.com/cropped/2024_06_04_e23d725cd2c5ceb81650g-2.jpg?height=320&width=404&top_left_y=518&top_left_x=858)\n\nWe measured $A B=1 \\mathrm{~cm}$ and $B C=5 \\mathrm{~cm}$. Calculate the width of the ribbon.",
        "answer": "\\frac{4}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 78166,
        "difficulty": null
      },
      {
        "id": "d0b54bbf7474b4dcc0ab23ac4a1416f3b65e7f89c8766b289cab2daea70a17c6",
        "domain": "numina",
        "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17061,
        "difficulty": null
      },
      {
        "id": "05a18529f00447718a79f90471fb8d0b01453ea67483032c39848f88103527ac",
        "domain": "numina",
        "question": "2.165. $\\frac{t^{2}-t-6-(t+3) \\sqrt{t^{2}-4}}{t^{2}+t-6-(t-3) \\sqrt{t^{2}-4}} ; t>2$.",
        "answer": "-\\sqrt{\\frac{+2}{-2}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47844,
        "difficulty": null
      },
      {
        "id": "8a058488857f00af979f9fab57ba01bb6f504804bce0310171de26ba3ab3677b",
        "domain": "numina",
        "question": "A4. A circle with radius 2 rolls around a regular pentagon with side 4. During this process, the circle remains pressed against the pentagon until it returns to its initial position.\n\nCalculate the area of the region that is covered by the circle during this revolution.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_6ccf422d4ecbc0dab674g-1.jpg?height=448&width=462&top_left_y=1666&top_left_x=1294)",
        "answer": "16(5+\\pi)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 85942,
        "difficulty": null
      },
      {
        "id": "efd511984f30770d0288d95aa09a33248a9779b2cfa4810659f28c42a14d2a0f",
        "domain": "numina",
        "question": "[Law of Cosines] [Law of Sines]\n\nFind the radius of the circumcircle of a triangle with sides 5 and 8 and an angle of $60^{\\circ}$ between them.",
        "answer": "\\frac{7\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59259,
        "difficulty": null
      },
      {
        "id": "85fdf5e8e5d150733ff76746297ac468b08f14c1decacdc6e4e7cee801c76eab",
        "domain": "numina",
        "question": "7. From all four-digit numbers where each digit is unique and the sum of the digits is 10, if two numbers are randomly selected, what is the probability that 2017 is selected?",
        "answer": "\\frac{1}{48}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 100307,
        "difficulty": null
      },
      {
        "id": "84edfc33ca0fc18849c6e35012c4b1fa8d72737e250c99f532f7b7ed7282042c",
        "domain": "numina",
        "question": "116. How many lines can divide a plane into 5 parts?",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44693,
        "difficulty": null
      },
      {
        "id": "a48cf609ef89f92bc749552cbc8bb7f238fcfe3a55394c5b9a333eb73866d873",
        "domain": "numina",
        "question": "6. A real estate agent is trying to sell the last apartment in a building for $482,100 \\mathrm{kn}$, which was the price of the penultimate apartment, and by doing so, the average price of the apartments he sold in that building would be $519,500 \\mathrm{kn}$. However, due to market saturation, he sells the apartment for only $451,500 \\mathrm{kn}$, making the average price of the apartments he sold in that building $517,700 \\mathrm{kn}$. How many apartments in that building has the real estate agent sold?",
        "answer": "17",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87020,
        "difficulty": null
      },
      {
        "id": "beb18f1828cb4c8942210cfaf74d8eae3b9dd259932e17086b6dbaea7ef4fac6",
        "domain": "numina",
        "question": "2. (17 points) There are ten weights of different weights, each weighing an integer number of grams. It is known that the weight of the lightest and heaviest weight differs by 9 grams. One weight is lost. Find its weight if the total weight of the remaining weights is 2022 grams.",
        "answer": "223",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4279,
        "difficulty": null
      },
      {
        "id": "343393ee22ead5db97cf16be4c79af19e80164f04e1fa4243bdab50f73460f14",
        "domain": "numina",
        "question": "4. Masha bought 2021 weights of pairwise distinct masses. Now Masha places one weight on each pan of a two-pan balance (weights placed on the balance previously are not removed). Each time the balance is in equilibrium, Masha rejoices. What is the maximum number of times she can find a reason to be happy",
        "answer": "673",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21640,
        "difficulty": null
      },
      {
        "id": "56acd928255deb7ea12ca1648771cd01dc0390c5a6dfa7bed8eea909bc7f8cb9",
        "domain": "numina",
        "question": "On each wall of a regular octahedron, one of the numbers $1,2,3,4,5,6,7$ and 8 is written, with different numbers on different walls. For each wall, Jarda determined the sum of the number written on it and the numbers of the three adjacent walls. Thus, he obtained eight sums, which he also added together.\n\nWhat values can this final sum take?\n\n(J. Zhouf)\n\nHint. How many times is each number counted in the total sum?",
        "answer": "144",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 76188,
        "difficulty": null
      },
      {
        "id": "1f332919b47ae68bd89a5c03b18f9d2236d63841940f7cdca0cc5038d6580429",
        "domain": "numina",
        "question": "3. Determine how many times the number $((2014)^{2^{2014}}-1)$ is larger than the number written in the following form: $\\left.\\left.\\left((2014)^{2^{0}}+1\\right) \\cdot\\left((2014)^{2^{1}}+1\\right) \\cdot\\left((2014)^{2^{2}}+1\\right) \\cdot \\ldots \\quad \\cdot\\right)^{2^{2013}}+1\\right)$.",
        "answer": "2013",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 8567,
        "difficulty": null
      },
      {
        "id": "f16ca5689e0044adc0545302d3edb1b5e2da287daefd392c81b201077fbb8964",
        "domain": "numina",
        "question": "8. Find all values of the parameter $a$ for which the system\n\n$$\n\\left\\{\\begin{array}{l}\ny=\\frac{|x|-1}{|x+1|}-2 \\\\\n4|x-1.5-a|+|y-1-2a|=1\n\\end{array}\\right.\n$$\n\nhas a unique solution. In your answer, specify the smallest of all the obtained values of the parameter $a$.",
        "answer": "-1.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10083,
        "difficulty": null
      },
      {
        "id": "147b64c207cd6f4593b94127ba1387f50121d0170931bd245c9314b33a97f289",
        "domain": "numina",
        "question": "12th Balkan 1995 Problem 1 Define a n by a 3 = (2 + 3)/(1 + 6), a n = (a n-1 + n)/(1 + n a n-1 ). Find a 1995 . Solution",
        "answer": "\\frac{1991009}{1991011}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 148748,
        "difficulty": null
      },
      {
        "id": "7e0ae242b4f2ef1054640e086a99aaa169fcef1093124d730204324b825c59cc",
        "domain": "numina",
        "question": "We have a lighter and two pieces of string that each burn for one hour when lit at one end (but not necessarily at a constant speed). How can we measure three-quarters of an hour?",
        "answer": "\\frac{3}{4}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 84915,
        "difficulty": null
      },
      {
        "id": "1cfd782d80cfe0849bb06642fbcba6b6f70800465139308b553bb0373582bbaa",
        "domain": "numina",
        "question": "The sum of the lengths of all of the edges of rectangular prism $A B C D E F G H$ is 24 . If the total surface area of the prism is 11, determine the length of the diagonal $A H$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_22929074a9efc78b4168g-2.jpg?height=380&width=542&top_left_y=279&top_left_x=846)",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194131,
        "difficulty": null
      },
      {
        "id": "62deb3845e7109230c841502c4f86cce7ef9eb4866877a4ac72d763473c34d8f",
        "domain": "numina",
        "question": "3. A group of toddlers in a kindergarten has 90 teeth in total. Any two toddlers together do not have more than 9 teeth. What is the minimum number of toddlers that can be in the group?",
        "answer": "23",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 21567,
        "difficulty": null
      },
      {
        "id": "10090029d89fa002166d3535e7e886fe7335c28f70f79c3c833edebffa15f380",
        "domain": "numina",
        "question": "What digit can the positive integer $n \\geq 3$ end with, if $n+n^{2}+\\ldots+n^{2 n-3}-4$ is a prime number?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171185,
        "difficulty": null
      },
      {
        "id": "209b2e28c8a0c4b99ee78e55e5320962791b7d96aa58056e863897aec70ece29",
        "domain": "numina",
        "question": "1. Given the sets $M=\\{x, x y, \\lg (x y)\\}$ and $N=\\{0,|x|, y\\}$, and $M=N$, then the value of $\\left(x+\\frac{1}{y}\\right)+$ $\\left(x^{2}+\\frac{1}{y^{2}}\\right)+\\left(x^{3}+\\frac{1}{y^{3}}\\right)+\\cdots+\\left(x^{2001}+\\frac{1}{y^{2(101)}}\\right)$ is $\\qquad$",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132727,
        "difficulty": null
      },
      {
        "id": "caddd862859c1a92c4b8a82461d7ab918a8e3e448bb827ab6383b1690d317042",
        "domain": "numina",
        "question": "7.1. In a row, the numbers $\\sqrt{7.301}, \\sqrt{7.302}, \\sqrt{7.303}, \\ldots, \\sqrt{16.002}, \\sqrt{16.003}$ are written (under the square root - consecutive terms of an arithmetic progression with a common difference of 0.001). Find the number of rational numbers among the listed ones.",
        "answer": "13",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17306,
        "difficulty": null
      },
      {
        "id": "ec0da5b29c3e0cc09593d513913c911f793c89b783516d80dbd9481d8eb03609",
        "domain": "numina",
        "question": "4. Let $a, b, c, d, e$ be positive integers. Their sum is 2018. Let $M=\\max (a+b, b+c, c+d, d+e)$. Find the smallest possible value of $M$.",
        "answer": "673",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4068,
        "difficulty": null
      },
      {
        "id": "4f3b2bfa10e0fc961c2c9466280ee165cf96b40392b8eae3c20aa81bb0f7b1b6",
        "domain": "numina",
        "question": "I2.2 If the height of a cylinder is doubled and the new radius is $h$ times the original, then the new volume is $k$ times the original. Find $k$.",
        "answer": "32",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152567,
        "difficulty": null
      },
      {
        "id": "b2d3e301ad7f79b132e55538f54909ff1bd731cf86880558c68d2c5aae3f2198",
        "domain": "numina",
        "question": "Starting from a number, you can either cube it or divide it by 8. Starting from the number 2 and using these two operations, can you reach 64? And $2^{2011}$?",
        "answer": "2^{2011}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78388,
        "difficulty": null
      },
      {
        "id": "1553a6bff73dd04774891c6faf8da96019435e5c34422e843592d4a094cb0295",
        "domain": "numina",
        "question": "(7) An ant starts from vertex $A$ of the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, and travels along the surface of the prism to reach vertex $C_{1}$. The shortest distance is 6. Then, the maximum volume of the rectangular prism is $\\qquad$ .",
        "answer": "12\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127316,
        "difficulty": null
      },
      {
        "id": "08327624833daccfacb9afb961643a4891b79800dcac1b8342c2c6c129af9ecf",
        "domain": "numina",
        "question": "Given the hyperbola $\\frac{x^{2}}{a}-\\frac{y^{2}}{a}=1$ with its right focus at $F$, any line passing through $F$ intersects the right branch of the hyperbola at $M, N$. The perpendicular bisector of $M N$ intersects the $x$-axis at $P$. When $a$ takes any real number, the value of $\\frac{\\vdots F P}{|M N|}$ is $\\qquad$",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129640,
        "difficulty": null
      },
      {
        "id": "045e8b99e0ffce227b7ceab547a88cd0175edaababe835a51accbce8ba5a59ab",
        "domain": "numina",
        "question": "25. There are 8 items with different masses, each of which is an integer number of grams and does not exceed 15 grams. Xiao Ping wants to find the item with the greatest mass using the fewest number of weighings on a balance. He used the following method:\n(1) Divide the 8 items into 2 groups, each with 4 items, and compare the masses of these 2 groups;\n(2) Take the 4 items from the heavier group and divide them into 2 groups, each with 2 items, and compare their masses;\n(3) Take the 2 items from the heavier group and divide them into 1 item each, and take the heavier one.\nXiao Ping did not get a balanced result on the balance in all 3 weighings, and finally obtained one item.\nHowever, the item he found is actually the 5th heaviest among the 8 items.\nQuestion: What is the mass of the item Xiao Ping found? And what is the mass of the second heaviest item?",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 120126,
        "difficulty": null
      },
      {
        "id": "085313bdd1df077e7e4b4b9c7e4514ede97e785ceef2052079c2cffbafbf4346",
        "domain": "numina",
        "question": "88. Divide the numbers $1, 2, 3, 4, 5, 6, 7, 8$ into three groups, and calculate the sum of the numbers in each group. If these three sums are all different, and the largest sum is twice the smallest sum, then the smallest sum is $\\qquad$.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 95555,
        "difficulty": null
      },
      {
        "id": "c2146cfbd95dbd558af1c33d4126d9ba90367c62c383305b5f6c7c601128e052",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\frac{1+\\cos (x-\\pi)}{\\left(e^{3 x}-1\\right)^{2}}$",
        "answer": "\\frac{1}{18}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 45973,
        "difficulty": null
      },
      {
        "id": "c3cc32f42a8bc0027ec18b4df17902bdfa9d14395c3c3707684e7f8a25c5e343",
        "domain": "numina",
        "question": "7.1. Come up with at least one three-digit PAU number (all digits are different) such that $(П+\\mathrm{A}+\\mathrm{У}) \\times \\Pi \\times \\mathrm{A} \\times \\mathrm{Y}=300$ (it is sufficient to provide one example)",
        "answer": "235",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9622,
        "difficulty": null
      },
      {
        "id": "07205b3a2b6dc3a881b01622eaf5e0f7213fa602bf8938474192af82a24c0e92",
        "domain": "numina",
        "question": "Shapovalov A.B.\n\n55 boxers participated in a tournament with a \"loser leaves\" system. The fights proceeded sequentially. It is known that in each match, the number of previous victories of the participants differed by no more than 1. What is the maximum number of fights the tournament winner could have conducted?",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 70595,
        "difficulty": null
      },
      {
        "id": "ee1004d59fd0a81afb4e80a47a711585cf0c1c2081a4d4f9575b93dd7826a588",
        "domain": "numina",
        "question": "(9) In the Cartesian coordinate system, the \"rectangular distance\" between points $P\\left(x_{1}, y_{1}\\right)$ and $Q\\left(x_{2}, y_{2}\\right)$ is defined as $d(P, Q)=\\left|x_{1}-x_{2}\\right|+\\left|y_{1}-y_{2}\\right|$. If the \"rectangular distance\" from $C(x, y)$ to points $A(1,3)$ and $B(6,9)$ is equal, where real numbers $x$ and $y$ satisfy $0 \\leqslant x \\leqslant 10,0 \\leqslant y \\leqslant 10$, then the sum of the lengths of all the loci of points $C$ that satisfy the condition is $\\qquad$.",
        "answer": "5(\\sqrt{2}+1)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129397,
        "difficulty": null
      },
      {
        "id": "e2c971fdab656902bcf9e8767a54b25b07b9979797c8530d1127568dc0bf43fa",
        "domain": "numina",
        "question": "2. Find the sum of the first twelve terms of an arithmetic progression if its fifth term $a_{5}=1$, and the seventeenth term $a_{17}=18$.",
        "answer": "37.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19201,
        "difficulty": null
      },
      {
        "id": "c5f15a96c0bd3d39feb059496ca73724f2950df54a6cbb429814392946a237f4",
        "domain": "numina",
        "question": "2. Four cars $A, B, C$, and $D$ start simultaneously from the same point on a circular track. $A$ and $B$ drive clockwise, while $C$ and $D$ drive counterclockwise. All cars move at constant (but pairwise different) speeds. Exactly 7 minutes after the start of the race, $A$ meets $C$ for the first time, and at the same moment, $B$ meets $D$ for the first time. After another 46 minutes, $A$ and $B$ meet for the first time. After how much time from the start of the race will all the cars meet for the first time?",
        "answer": "371",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11722,
        "difficulty": null
      },
      {
        "id": "a5d747164c9b746c25fd39e040cb4cd5d7c147f44d682a2fbec2003e97419366",
        "domain": "numina",
        "question": "3 、The product of two positive integers is 100, neither of these positive integers contains the digit 0, then the sum of these two positive integers is",
        "answer": "29",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 110105,
        "difficulty": null
      },
      {
        "id": "62092a79d2073e14cf806bdec4d994db1dc21787d549f7d77408d3e86b9c7699",
        "domain": "numina",
        "question": "(4) (50 points) Let $A$ be a $3 \\times 9$ grid, with each small cell filled with a positive integer. A sub-grid of $A$ of size $m \\times n (1 \\leqslant m \\leqslant 3, 1 \\leqslant n \\leqslant 9)$ is called a \"good rectangle\" if the sum of all its numbers is a multiple of 10. A $1 \\times 1$ cell in $A$ is called a \"bad cell\" if it is not contained in any \"good rectangle\". Find the maximum number of \"bad cells\" in $A$.",
        "answer": "25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132968,
        "difficulty": null
      },
      {
        "id": "6badefa40444d5400ab94ad5d70db54364aed9a63931e03005aeef462a1f688a",
        "domain": "numina",
        "question": "6.3. In the park, all bicycle paths run from north to south or from west to east. Petya and Kolya simultaneously started from point $A$ and rode their bicycles at constant speeds: Petya - along the route $A-B-C$, Kolya - along the route $A-D-E-F-C$ (see fig.), and both spent 12 minutes on the road. It is known that Kolya rides 1.2 times faster than Petya. How much time did he spend on the segment $D E$? The scale in the figure is not preserved.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_f265a9605ca3fc80eb59g-1.jpg?height=224&width=408&top_left_y=1437&top_left_x=885)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13019,
        "difficulty": null
      },
      {
        "id": "f2d7a990e24702f192a18f4ac138981c4673b54d9a38bbe94c201de4d5ce4c4d",
        "domain": "numina",
        "question": "15.1 If $144^{p}=10,1728^{q}=5$ and $a=12^{2 p-3 q}$, find the value of $a$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152886,
        "difficulty": null
      },
      {
        "id": "5e8cee9f55e3ba43296c5153cbe4436c12d6d627723f40dc9d991c3125f73d99",
        "domain": "numina",
        "question": "During a summer vacation, it rained 7 times in the morning or afternoon. If it rained in the morning, it did not rain in the afternoon. In total, there were 5 rainless mornings and 6 rainless afternoons. How long did the vacation last?",
        "answer": "9",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 168423,
        "difficulty": null
      },
      {
        "id": "e8fd4bd5f08f8735709a83b7e39a6b86c8c21059677b6487def70050aa9501c9",
        "domain": "numina",
        "question": "Task 3. Determine the number of:\n\na) two-digit\n\nb) three-digit numbers\n\nthat are written only with even digits.",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 188785,
        "difficulty": null
      },
      {
        "id": "451c4cef07dd8ec9c29a06764ff3adf945be0a76c6073fc601af2d023bf8875b",
        "domain": "numina",
        "question": "7.4. Arseny sat down at the computer between 4 and 5 PM, when the hour and minute hands were pointing in opposite directions, and got up from it on the same day between 10 and 11 PM, when the hands coincided. How long did Arseny sit at the computer?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 9969,
        "difficulty": null
      },
      {
        "id": "3c4a97e773599b742c632075cba0b63f3311f4f4e3f44985a3493369222576f9",
        "domain": "numina",
        "question": "Example 9. On five identical cards, letters are written: on two cards $l$, on the other three $i$. These cards are randomly laid out in a row. What is the probability that the word \"lilii\" will be formed?",
        "answer": "0.1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32904,
        "difficulty": null
      },
      {
        "id": "5e9efd33aff16dd3036f56e9559cd7a739d6b0a03cc79b40df1de7073a46a4fd",
        "domain": "numina",
        "question": "84. The probability of successfully performing an exercise for each of two athletes is 0.5. The athletes perform the exercise in turns, with each making two attempts. The one who performs the exercise first wins the prize. Find the probability of the athletes winning the prize.",
        "answer": "0.9375",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32303,
        "difficulty": null
      },
      {
        "id": "f0df1e51f99fecdc11dc67e2170f12c0cd1e7e9987d33d5e585ee622e417dfad",
        "domain": "numina",
        "question": "4. How many orderings $\\left(a_{1}, \\ldots, a_{8}\\right)$ of $(1,2, \\ldots, 8)$ exist such that $a_{1}-a_{2}+a_{3}-a_{4}+a_{5}-a_{6}+a_{7}-a_{8}=0$ ?",
        "answer": "4608",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151455,
        "difficulty": null
      },
      {
        "id": "511dbca9bde8c1b5009d8479c7ac49356c4f27e303a0e9dd293b6ad0b8da202c",
        "domain": "numina",
        "question": "Task B-3.4. A group of children found a wooden board in the shape of a quadrilateral and decided to use it for the game \"pick-a-point\". Since the target was of an unusual shape, they had to adapt the game rules. They asked for advice from Mark's older brother, a good mathematician. He measured, calculated, and wrote something... and came to an important conclusion. He established that there is a point $O$ equally distant from all sides of the quadrilateral. Then he divided the board into areas by drawing lines from point $O$ to all vertices $A, B, C, D$. He assigned points to the areas inversely proportional to their area. Which area yields the most points and why? Mark's brother added a condition that points are only awarded if the distance from the hit point $T$ to point $O$ is not greater than the distance from point $O$ to any side of the board. Determine the ratio of the area of the part of the board that does not yield points to the area of the part that does yield points.\n\nMark's brother measured these values $|A B|=5 \\mathrm{dm},|B C|=3 \\mathrm{dm},|A D|=4 \\mathrm{dm}$, $|\\angle B A D|=60^{\\circ}$.",
        "answer": "\\frac{7}{10\\pi}(5\\sqrt{3}-\\sqrt{5})-1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87232,
        "difficulty": null
      },
      {
        "id": "5ba6e84cf96b83bff404ede5e9d42485aa7698e1b4eb6e6475cca9b6e555ad81",
        "domain": "numina",
        "question": "11.2. On the board, there are 4 numbers. Vasya multiplied the first of these numbers by $\\sin \\alpha$, the second - by $\\cos \\alpha$, the third - by $\\operatorname{tg} \\alpha$, and the fourth - by $\\operatorname{ctg} \\alpha$ (for some angle $\\alpha$) and obtained a set of the same 4 numbers (possibly in a different order). What is the maximum number of different numbers that could have been written on the board?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14753,
        "difficulty": null
      },
      {
        "id": "26ce21dc8a3b23962fb93262dce29039cbfdc8eaae139b24fbe85960233de2e8",
        "domain": "numina",
        "question": "[ Arithmetic. Mental calculation, etc.]\n\nA customer walks into a store, picks out an item costing 20 rubles, and gives the seller a hundred-ruble note. The seller looks - no change. He goes to the neighboring department, gets the hundred-ruble note changed. He gives the customer the item and the change. The customer leaves. Suddenly, the seller from the neighboring department comes over, bringing the hundred-ruble note. It's a counterfeit! Our seller gives him his own hundred-ruble note. How much did our poor seller lose in the end?",
        "answer": "100",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 69080,
        "difficulty": null
      },
      {
        "id": "c8d7a7850be998258b318f66f28b206fc2c4122e26d5a4a279b9cb0144a34ab5",
        "domain": "numina",
        "question": "3. Solve the system of equations:\n\n$$\n\\left\\{\\begin{array}{l}\n2 x^{2}+3 y+5=2 \\sqrt{2 z+5} \\\\\n2 y^{2}+3 z+5=2 \\sqrt{2 x+5} \\\\\n2 z^{2}+3 x+5=2 \\sqrt{2 y+5}\n\\end{array}\\right.\n$$",
        "answer": "-0.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19799,
        "difficulty": null
      },
      {
        "id": "6d3ce26eae91d9fbab888e605be7278ea837c29ec7b63c08de64e06d8183af84",
        "domain": "numina",
        "question": "8. [6] Let $A B C$ be an equilateral triangle with side length 2 , and let $\\Gamma$ be a circle with radius $\\frac{1}{2}$ centered at the center of the equilateral triangle. Determine the length of the shortest path that starts somewhere on $\\Gamma$, visits all three sides of $A B C$, and ends somewhere on $\\Gamma$ (not necessarily at the starting point). Express your answer in the form of $\\sqrt{p}-q$, where $p$ and $q$ are rational numbers written as reduced fractions.",
        "answer": "\\sqrt{\\frac{28}{3}}-1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150954,
        "difficulty": null
      },
      {
        "id": "bdd2d6a2bc15e24d2fd9f60a1448add2ccf9440fbf94ec05efd1cc50c9e7fcbc",
        "domain": "numina",
        "question": "7. In the tetrahedron $S A B C$, $\\triangle A B C$ is an equilateral triangle. The projection of point $A$ onto the plane $S B C$ is $H$, which is the orthocenter of $\\triangle S B C$, and the dihedral angle $H-A B-C$ has a plane angle of $30^{\\circ}$. Given $S A=2 \\sqrt{3}$, the volume of the tetrahedron $S A B C$ is $\\qquad$.",
        "answer": "\\frac{9}{4}\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113951,
        "difficulty": null
      },
      {
        "id": "52e8f285c619777218a99c4c58cd7ccc7f2d4914ffa75a41cabdefcd9ee9fb4b",
        "domain": "numina",
        "question": "9. Let $B$ be a point on the circle centred at $O$ with diameter $A C$ and let $D$ and $E$ be the circumcentres of the triangles $O A B$ and $O B C$ respectively. Given that $\\sin \\angle B O C=\\frac{4}{5}$ and $A C=24$, find the area of the triangle $B D E$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160634,
        "difficulty": null
      },
      {
        "id": "74c0da87c8a835449a6f9740b6f752a91eaf8227d1799b309778ca630d8611d5",
        "domain": "numina",
        "question": "11. The town of Coppersville uses some special coins. Edward has five 5 p coins, four $4 p$ coins, three $3 \\mathrm{p}$ coins, two $2 \\mathrm{p}$ coins and one $1 \\mathrm{p}$ coin. The smallest amount he cannot pay with his coins alone is $W$ p.\nWhat is the value of $W$ ?",
        "answer": "56",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156361,
        "difficulty": null
      },
      {
        "id": "54a84ea07fbeba93e24ae34a9b0f04d5970d7a0877fc42120adc64df24efc25f",
        "domain": "numina",
        "question": "B3. Solve the irrational equation $\\sqrt{x^{2}+4}-\\sqrt{4 x}=0$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163836,
        "difficulty": null
      },
      {
        "id": "9fb1c4f1f96d8163ac517137231e8b526707fb154870ad05671c5a817ef6dbf9",
        "domain": "numina",
        "question": "4  [ Proof by contradiction $]$\n\nInside an isosceles right triangle $ABC$ with hypotenuse $AB$, a point $M$ is taken such that angle $MAB$ is $15^{\\circ}$ greater than angle $MAC$, and angle $MCB$ is $15^{\\circ}$ greater than angle $MBC$. Find angle $BMC$.",
        "answer": "150",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58558,
        "difficulty": null
      },
      {
        "id": "e20ceec0370ac5b43672009a82ba0173106134e860760166ec754d8017f2628c",
        "domain": "numina",
        "question": "5. Given that $a, b$ are positive numbers, and $\\frac{a+b}{a^{2}+a b+b^{2}}=\\frac{4}{49}$, then the value of $a+b$ is",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140593,
        "difficulty": null
      },
      {
        "id": "be1ee7c47f2d15edbd93ff10c5fe820606921b59da8aee61d57be17fff534684",
        "domain": "numina",
        "question": "7、Convert the fraction $\\frac{5}{7}$ into a decimal, then the sum of all digits from the 1st to the 100th position after the decimal point is $\\qquad$\n\nConvert the fraction $\\frac{5}{7}$ into a decimal, then the sum of all digits from the 1st to the 100th position after the decimal point is $\\qquad$",
        "answer": "446",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122568,
        "difficulty": null
      },
      {
        "id": "37945fb37cdda33c83f529f734525446ec435459b88b3e5761f77f401c4b8c80",
        "domain": "numina",
        "question": "8. In a unit square, if three of its four sides are colored with three different colors, then the unit square is called \"colorful\". As shown in the figure, a $1 \\times 3$ grid table contains 10 unit line segments in its grid lines. Now, we need to color these 10 line segments, with each line segment being colored red, yellow, or blue, such that all three unit squares are colorful. The number of such coloring ways is $\\qquad$ (answer in numerical form).",
        "answer": "5184",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 120636,
        "difficulty": null
      },
      {
        "id": "e1787247ecfe3e35c61986ebe5986f383d24699a0e47ed20b81526977235ea7d",
        "domain": "numina",
        "question": "## Problem II - 5\n\nThe natural numbers 22, 23, and 24 have the following property: the exponents of the prime factors in their factorization are all odd:\n\n$$\n22=2^{1} \\cdot 11^{1} ; \\quad 23=23^{1} ; \\quad 24=2^{3} \\cdot 3^{1}\n$$\n\nWhat is the largest number of consecutive natural numbers that can have this property? Justify your answer.",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86221,
        "difficulty": null
      },
      {
        "id": "d22461f152bdb8a49218d52077b5bb831b001a2ff18d0975cd557100a1742943",
        "domain": "numina",
        "question": "2. Two players take turns to write natural numbers on a board. The rules forbid writing numbers greater than $\\mathrm{p}$ and also divisors of previously written numbers. The player who has no move loses.\nDetermine which player has a winning strategy for $\\mathrm{p}=10$ and describe this strategy.",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153908,
        "difficulty": null
      },
      {
        "id": "317a347fe62830ba542949d042328632749b27a59ae85481ffc82b127f40578c",
        "domain": "numina",
        "question": "1. Given vectors $\\overrightarrow{A P}=(1, \\sqrt{3}), \\overrightarrow{P B}=(-\\sqrt{3}, 1)$, then the angle between vector $\\overrightarrow{A P}$ and $\\overrightarrow{A B}$ is $\\qquad$",
        "answer": "\\frac{\\pi}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 117150,
        "difficulty": null
      },
      {
        "id": "396c2fad4523b70a801dd82f52303007b335f20a823a3dce579de0232d4dc7d9",
        "domain": "numina",
        "question": "## Task A-1.2. (4 points)\n\nLet $a$ be a real number. Determine the sum of all three solutions of the equation\n\n$$\nx^{3}-a^{2} x+a x-x+a^{2}-a=0\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87454,
        "difficulty": null
      },
      {
        "id": "5d770b7e2979bd767afd4772c60fec2bdf62b496b71ce754873d03be7526090a",
        "domain": "numina",
        "question": "7. [7] Suppose $a$ and $b$ be positive integers not exceeding 100 such that\n$$\na b=\\left(\\frac{\\operatorname{lcm}(a, b)}{\\operatorname{gcd}(a, b)}\\right)^{2} .\n$$\n\nCompute the largest possible value of $a+b$.",
        "answer": "78",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152023,
        "difficulty": null
      },
      {
        "id": "c2b4515295017f9372ad10da0d68ab793b21af9d4c9fcadd8be71f89f7808e3c",
        "domain": "numina",
        "question": "The unit volume regular tetrahedron, $T$, is magnified by a factor of two from a point $P$ on one of its symmetry axes. The volume of the common part of the original and the magnified tetrahedron is $1 / 8$. How far is the center of $T$ from $P$? (H)",
        "answer": "\\frac{5}{6}\\sqrt[6]{243}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 164930,
        "difficulty": null
      },
      {
        "id": "d0d10932262a61f40c426793d51122e077d2506021104568a6155b34f61f00f0",
        "domain": "numina",
        "question": "2. Half-year grade. By the end of the half-year, Vasya Petrov had the following grades in mathematics in his journal: $4,1,2,5,2$. Before assigning the half-year grade, the math teacher told Vasya:\n\n- Vasya, you can choose the method to determine your half-year grade. I offer two options. Method A: the arithmetic mean of the current grades, rounded to the nearest whole number. Method B: the median of the current grades.\n\nThe best method for Vasya is the one that gives him the highest grade for the half-year.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_853fe0fb72821c0a3a48g-10.jpg?height=328&width=534&top_left_y=1912&top_left_x=1366)\n\na) (from 6th grade. 1 point). Which method is the best for Vasya?\n\nb) (from 6th grade. 2 points). Then the teacher thought and added:\n\n- Keep in mind, Vasya, that if you manage to choose the best method for yourself, I will add two more fives to your journal, and then calculate the half-year grade.\n\nProve that under these conditions, Method A is not the best for Vasya.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10951,
        "difficulty": null
      },
      {
        "id": "054ddec60f272f826576dda809bc678406bb25e3e1af3e05ac1500fef8295ee8",
        "domain": "numina",
        "question": "6. Let $g(x)$ be a strictly increasing function defined for all $x \\geq 0$. It is known that the range of $t$ satisfying\n$$\n\\mathrm{g}\\left(2 t^{2}+t+5\\right)<\\mathrm{g}\\left(t^{2}-3 t+2\\right)\n$$\nis $b<t<a$. Find $a-b$.",
        "answer": "2",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 160959,
        "difficulty": null
      },
      {
        "id": "5c821e9ffb384480a097b3df1639c09c4ea81d62d6d5ac18daaf1f898ce9eda8",
        "domain": "numina",
        "question": "7.1. In a hotel, rooms are arranged in a row in order from 1 to 10000. Masha and Alina checked into the hotel in two different rooms. The sum of the room numbers they are staying in is 2022, and the sum of the room numbers of all the rooms between them is 3033. In which room is Masha staying, if her room has a lower number than Alina's room?",
        "answer": "1009",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16861,
        "difficulty": null
      },
      {
        "id": "e3b975792817f9fb4b4ad76f919dde2529470241cd072d3d9340be8952711055",
        "domain": "numina",
        "question": "13.076. The collection of corn from the fields of a livestock farm was 4340 centners. The following year, it is planned to obtain 5520 centners of corn by increasing the area by 14 hectares and increasing the yield by 5 centners per hectare. Determine the area occupied by corn and the yield in centners per hectare (the yield was less than 40 centners per hectare).",
        "answer": "124",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51575,
        "difficulty": null
      },
      {
        "id": "44322837a71f1e8e2dbfcc41e42076a05da0923987585bb2f2fe749f04970bb8",
        "domain": "numina",
        "question": "A regular hexagon $S$ has an inscribed triangle $H$ such that one side of $H$ is parallel to one side of $S$. What is the maximum possible ratio of the area of $H$ to the area of $S$?",
        "answer": "\\frac{3}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166774,
        "difficulty": null
      },
      {
        "id": "730cc06dd8779da8fd9bdca2f2a362f581e2ce395428db9a2aa63f1e144c3c62",
        "domain": "numina",
        "question": "5.086 In how many ways can the players of two football teams be arranged in a single row so that no two players from the same team stand next to each other?",
        "answer": "2\\cdot(11!)^2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 52305,
        "difficulty": null
      },
      {
        "id": "136b1c36d892fd3a9d8e64ea80ade50fa8f92ec2c4f0ed7f9ac3d2ad9eb33dba",
        "domain": "numina",
        "question": "The director of the school decided to take a photo of the 2008 graduates. He arranged the students in parallel rows, all with the same number of students, but this arrangement was too wide for the camera's field of view. To solve this problem, the director noticed that it was enough to remove one student from each row and place them in a new row. This arrangement did not please the director because the new row had 4 students fewer than the others. He then decided to remove one more student from each row and place them in the new row he created, and found that this way all the rows had the same number of students, and finally took the photo. How many students appeared in the photo?",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 73905,
        "difficulty": null
      },
      {
        "id": "899b1855b58c6d211515be3b2a78b96f14678e33ceea25c0fe00b9146da536d0",
        "domain": "numina",
        "question": "13. On the Island of Knights and Rascals, Knights tell the truth except when they are wrong, and Rascals always lie. During a meeting, 40 islanders sit around a large round table, and each one says: \"I am next to a Knight and a Rascal.\" Knowing that 3 Knights present are wrong, how many Knights are at the meeting?",
        "answer": "27",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181587,
        "difficulty": null
      },
      {
        "id": "e7458c487e9b8b84457bf46373602533f953da24eab4355b3eefbfc959da8f35",
        "domain": "numina",
        "question": "Problem 10.1. An equilateral triangle with a side of 10 is divided into 100 small equilateral triangles with a side of 1. Find the number of rhombi consisting of 8 small triangles (such rhombi can be rotated).\n![](https://cdn.mathpix.com/cropped/2024_05_06_d4986512fdb0330805a0g-32.jpg?height=280&width=582&top_left_y=1198&top_left_x=434)",
        "answer": "84",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 14026,
        "difficulty": null
      },
      {
        "id": "3a9a453854e09d84e84d3b7f9e1365f620cd46bf80b0be9551430abaae853e50",
        "domain": "numina",
        "question": "6. What can you buy for a ruble? Nine boxes of matches cost 9 rubles and some kopecks, while ten such boxes cost 11 rubles and some kopecks. How much does one box cost?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35556,
        "difficulty": null
      },
      {
        "id": "7a958520ce9ea318384eceee9fd587f8eea58d16212b585d0230b682967c8916",
        "domain": "numina",
        "question": "In a parallelogram, there are two circles. One of them, with a radius of 3, is inscribed in the parallelogram, and the second circle touches two sides of the parallelogram and the first circle. The distance between the points of tangency lying on the same side of the parallelogram is 3. Find the area of the parallelogram.",
        "answer": "\\frac{75}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65600,
        "difficulty": null
      },
      {
        "id": "74d65fd5590e6be986ed9492e985ebfc430c9010c7ae90402075998f0f2efc91",
        "domain": "numina",
        "question": "## 15. The Emir and His Oil\n\nMy emirate encompasses a vast desert, in the middle of which I have built my palace. It also includes territorial waters along the coastline, which I value greatly. The territorial waters are quite extensive (covering an area that reaches a third of the desert's area), and part of my oil field is located there. Knowing that the area of the desert with oil deposits is three times the area of the marine territory without oil and that one-seventh of my territory without oil is in the sea, could you determine exactly what fraction of my oil field is in the sea?",
        "answer": "\\frac{2}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 41993,
        "difficulty": null
      },
      {
        "id": "e00e29c87a7a3d14876f2ad9aeb16fffad867fd7bfaaa7d826838b391473fa68",
        "domain": "numina",
        "question": "21st Putnam 1960 Problem A6 A player throws a fair die (prob 1/6 for each of 1, 2, 3, 4, 5, 6 and each throw independent) repeatedly until his total score ≥ n. Let p(n) be the probability that his final score is n. Find lim p(n). Solution",
        "answer": "\\frac{2}{7}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 147937,
        "difficulty": null
      },
      {
        "id": "9440d1760a330732db08f7bb2113c70314cb2990cb5bffb865ad7217f16052c5",
        "domain": "numina",
        "question": "10.5. To a natural number $N$, the largest divisor of $N$ less than $N$ was added, and the result was a power of ten. Find all such $N$.\n\n(N. Agakhanov)",
        "answer": "75",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13144,
        "difficulty": null
      },
      {
        "id": "ac720cba446f5c09c4424fb5926b6d45cc2fb9eacda09c7cf7ba72f83aeeb632",
        "domain": "numina",
        "question": "72. Railway Network (I). There are five cities; no three of them lie on the same straight line. These cities need to be connected by a railway network consisting of four straight roads; at that, railway lines can be built one over another on viaducts.\n\nHow many such different railway networks exist?",
        "answer": "125",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 35400,
        "difficulty": null
      },
      {
        "id": "e00652278848e0370b5b04e0cbfbc6d7092e019d4b16a57c1f55abd9dad524a3",
        "domain": "numina",
        "question": "3. In city \"N\", there are 10 horizontal and 12 vertical streets, of which a pair of horizontal and a pair of vertical streets form the rectangular boundary of the city, while the rest divide it into blocks that are squares with a side length of 100 meters. Each block has an address consisting of two integers $(i ; j), i=1,2, . ., 9, j=1,2, \\ldots, 11-$ the numbers of the streets that bound it from below and from the left. Taxis transport passengers from one block to another, adhering to the following rules: 1) pick-up and drop-off can be made at any point on the boundary of the block at the passenger's request; 2) it is forbidden to enter inside the block; 3) transportation is carried out along the shortest path; 4) a fee of 1 coin is charged for every 100 meters traveled (rounding the distance to the nearest 100 meters in favor of the driver). How many blocks are there in the city? What is the maximum and minimum fare that the driver can charge the passenger for a ride from block $(7,1)$ to block $(2 ; 10)$ without violating the rules?",
        "answer": "99",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 2107,
        "difficulty": null
      },
      {
        "id": "3176a7d7a5b0c5718c8d6c8eeace0068f5ea8dcad51b30bce5038a69c103f59c",
        "domain": "numina",
        "question": "8. From $1,2, \\cdots, 20$ choose 5 different numbers, the probability that at least two of them are consecutive is",
        "answer": "\\frac{232}{323}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 112018,
        "difficulty": null
      },
      {
        "id": "c9a9b65de2176d3b7f52e370f7d61bbe1fe150b4653d4a3abdfdce94f399ad1c",
        "domain": "numina",
        "question": "1. In triangle ABC, the altitude to side BC, the perpendicular bisector of the arc $\\mathrm{ABC}$, and the perpendicular bisector of side $\\mathrm{AB}$ intersect at one point. Determine the measure of angle $\\mathrm{ABC}$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189243,
        "difficulty": null
      },
      {
        "id": "eef8f635fca0cdff109481ea9a8692444127325b5234f60e6c38531e0e528531",
        "domain": "numina",
        "question": "1. The numbers $62,63,64,65,66,67,68,69$ and 70 are divided by, in some order, the numbers $1,2,3,4,5,6,7,8$ and 9 , resulting in nine integers. The sum of these nine integers is $S$. What are the possible values of $S$ ?",
        "answer": "187",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154971,
        "difficulty": null
      },
      {
        "id": "97cc31b9e0f3f59f44f54d9f3852fe8e1240cdb407ff7508eeb345e5f63ed57d",
        "domain": "numina",
        "question": "4. Xiao Ming's mother went to the store to buy soap, the unit prices of brand A soap and brand B soap are 6 yuan and 9 yuan respectively. If Xiao Ming's mother used all her money to buy brand A soap, she could buy 2 more bars than if she used all her money to buy brand B soap, and there would be no money left. So, how much money did Xiao Ming's mother bring?",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127455,
        "difficulty": null
      },
      {
        "id": "81909e7578c95d73dcaf421632415efcaa5aa2208c9ce52e72070c91b4102bb0",
        "domain": "numina",
        "question": "Let $(x, y, z, t) \\in\\left(\\mathbb{N}^{*}\\right)^{4}$ be a quadruplet satisfying $x+y=z+t$ and $2 x y=z t$, and such that $x \\geqslant y$. Find the largest value of $m$ such that $m \\leqslant x / y$ for sure.",
        "answer": "3+2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 84377,
        "difficulty": null
      },
      {
        "id": "373e2d12241e732e056c644255813a20137049c3d4638a88607782d083f835db",
        "domain": "numina",
        "question": "5.1. Among all the simple fractions, where the numerator and denominator are two-digit numbers, find the smallest fraction greater than $\\frac{3}{4}$. In your answer, specify its numerator.",
        "answer": "73",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6460,
        "difficulty": null
      },
      {
        "id": "fa241f6b037980e626837d0d8ff6a2cc08491cadd2c2081e252cf4411284f414",
        "domain": "numina",
        "question": "[ Rectangles and squares. Properties and characteristics ] [ Congruent triangles. Criteria for congruence (other).]\n\nMitya is going to fold a square sheet of paper $ABCD$. Mitya calls a fold beautiful if side $AB$ intersects side $CD$ and the four resulting right triangles are congruent. Before this, Vanya chooses a random point $F$ on the sheet. Find the probability that Mitya can make a beautiful fold passing through point $F$.",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61069,
        "difficulty": null
      },
      {
        "id": "5a2f6f75970dafe58259635ca39bdf0424bacf779732db7c83e9b25309b6fe84",
        "domain": "numina",
        "question": "Calculate the remainder of the Euclidean division of $2022^{2023^{2024}}$ by 19.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 81705,
        "difficulty": null
      },
      {
        "id": "df854df6616b7754db683c609eef13788fba1786bcb3d28e6a1e9eab50885774",
        "domain": "numina",
        "question": "## Exercise 11.\n\n1) Alice wants to color the integers between 2 and 8 (inclusive) using $k$ colors. She wishes that if $m$ and $n$ are integers between 2 and 8 such that $m$ is a multiple of $n$ and $m \\neq n$, then $m$ and $n$ are of different colors. Determine the smallest integer $k$ for which Alice can color the integers $2,3, \\ldots, 8$ using $k$ colors.\n2) Alice wants to color the integers between 2 and 31 (inclusive) using $k$ colors. She wishes that if $m$ and $n$ are integers between 2 and 31 such that $m$ is a multiple of $n$ and $m \\neq n$, then $m$ and $n$ are of different colors. Determine the smallest integer $k$ for which Alice can color the integers $2,3, \\ldots, 31$ using $k$ colors.",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78121,
        "difficulty": null
      },
      {
        "id": "c079a0809ded418a2d5f303cd2ae8fad82ad8ed34c97736817078e2132d1093a",
        "domain": "numina",
        "question": "Determine the product of all positive integers $n$ for which $3(n!+1)$ is divisible by $2 n-5$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nHowever, it seems there is a note or an additional request in Chinese at the end of your message. If you need the Chinese text translated as well, please let me know! For now, I'll assume you only need the provided Dutch text translated to English. If that's correct, the translation is as above.",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 195346,
        "difficulty": null
      },
      {
        "id": "423ad9d038a078a73e869589c07a94f7c7542e9fae0672d9c6f747e3338e504f",
        "domain": "numina",
        "question": "1.5 When a piece of paper with digits is flipped upside down, the digits $0,1,8$ remain unchanged, the digits 6 and 9 swap, and other digits have no meaning when flipped. Find the number of nine-digit numbers that remain unchanged when the paper is flipped upside down.",
        "answer": "1500",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123392,
        "difficulty": null
      },
      {
        "id": "b03b2d89234aaf3c35c9a441fb7f2664387ecdd251e9932d43e83695a526cea6",
        "domain": "numina",
        "question": "5. In a football tournament where each team played against each other once, teams A, B, C, D, and E participated. For a win, a team received 3 points, for a draw 1 point, and for a loss 0 points. In the end, it turned out that teams A, B, C, D, and E each had 7 points. What is the maximum number of points that team $\\mathrm{E}$ could have?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12566,
        "difficulty": null
      },
      {
        "id": "ba0274984aeb54b4ebde4b7749bd0e75980d254b8c54844852e3ee9e7ded0b80",
        "domain": "numina",
        "question": "Task 1. Which of the numbers 723, 732, and 273 will decrease the most and by how much if in each of them the digits 7 and 3 swap places?",
        "answer": "396",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188929,
        "difficulty": null
      },
      {
        "id": "dbad4eb3c75079600f0d48d9de3050e2cc338a03e3107731ad9c4c6c314d2bd0",
        "domain": "numina",
        "question": "## Properties and characteristics of a parallelogram [Trigonometric ratios in a right triangle]\n\nOn the side $C D$ of parallelogram $A B C D$ with an obtuse angle at vertex $D$, an equilateral triangle $C D E$ is constructed such that points $A$ and $E$ lie on opposite sides of line $C D$. It is known that the distances from points $D$ and $E$ to line $B C$ are 3 and 8, respectively, and the distance from point $E$ to line $A B$ is 13. Find the area of parallelogram $A B C D$.",
        "answer": "28\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62253,
        "difficulty": null
      },
      {
        "id": "e81bbbd1473f9638b2c8c20f7f5cfaa81bf54a74a3afd8cea294c0ed0142068a",
        "domain": "numina",
        "question": "The ratio between the number of men and women in the city of Campo Verde is $\\frac{2}{3}$. The average age of men is 37 years and that of women is 42 years. What is the average age of the inhabitants of Campo Verde?",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73886,
        "difficulty": null
      },
      {
        "id": "e00578a172d87728d106e094b3590dc0c914b5aac5d7f0305ea9c6fa570cf2ef",
        "domain": "numina",
        "question": "a) A point $M$ moves around the circumcircle of a triangle, and for each of its positions, we consider the projections of $M$ onto the sides of the triangle. We find that the points where $M$ is such that all three projections fall on the extensions of the sides (or exactly at the vertices) cover $\\frac{8}{10}$ of the circumference of the circle. What is the largest angle of the triangle?\n\nb) A point $N$ moves around the incircle of a triangle. We find that the points where $N$ is such that one of the three projections (formed in the same way as before) falls on the extension of a side cover $\\frac{8}{10}$ of the circumference of the circle. What is the largest angle of the triangle? (It is sufficient to give the angle with an accuracy of $0.1^{\\circ}$.)",
        "answer": "162",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166557,
        "difficulty": null
      },
      {
        "id": "67f28c83d2825ad9d1a8f1aedc2fbc5eb7b4d837ed3c038b70011d4643f586c2",
        "domain": "numina",
        "question": "17. How many roots has the equation $\\sin x-\\log _{10} x=0$ ?",
        "answer": "3",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 154098,
        "difficulty": null
      },
      {
        "id": "1bca341cad043b6ac3ccdea8757eded323a205aacf676e3f722eec3f491b5f80",
        "domain": "numina",
        "question": "## 114. Math Puzzle $11 / 74$\n\nIn a numerically controlled machine tool, the electronics account for $40 \\%$ of the total price.\n\nHow much does the electronics cost in relation to the actual machine?",
        "answer": "67",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181794,
        "difficulty": null
      },
      {
        "id": "602febf206e6abc28ff46f37504e3fa57bbb171fb7c9a8ed10bbad5d3a9fe8fc",
        "domain": "numina",
        "question": "13.228. Two workers were assigned to manufacture a batch of identical parts. After the first worked for 2 hours and the second for 5 hours, it turned out that they had completed half of the entire work. Working together for another 3 hours, they found that they had 0.05 of the entire work left to complete. In what time interval can each of them, working separately, complete the entire work?",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49316,
        "difficulty": null
      },
      {
        "id": "d06e4c1008b850749528ac274485702951fa77c3d68dda4fedc3512390c9094e",
        "domain": "numina",
        "question": "3. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then the sum\n$$\nf\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+f\\left(\\frac{3}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)\n$$\n\nis equal to . $\\qquad$",
        "answer": "500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116142,
        "difficulty": null
      },
      {
        "id": "34cd3f98a4cc4005ba07431c825e5c796ce99d53eba753754104d517754743a4",
        "domain": "numina",
        "question": "10. (20 points) How many zeros does the number $4^{5^{6}}+6^{5^{4}}$ end with in its decimal representation?\n\n#",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19379,
        "difficulty": null
      },
      {
        "id": "ea253b94ee8c48bd40c9827a38a7cb1baa6654aa732a4220dd098d2de70ddda5",
        "domain": "numina",
        "question": "1. Joey is standing at a corner of the rectangular field shown. He walks the perimeter of the field 5 times. How many metres does Joey walk?",
        "answer": "1500",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 145836,
        "difficulty": null
      },
      {
        "id": "647b8ce93ae90b5a2cd453f9d976fdbb6649e9a4c896bec6953d597bb06ab133",
        "domain": "numina",
        "question": "$\\left.\\frac{\\text { [Inscribed and Circumscribed Circles ]}}{\\text { Rhombuses Properties and Characteristics }}\\right]$\n\nThe side of the rhombus is 8 cm, and the acute angle is $30^{\\circ}$. Find the radius of the inscribed circle.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57936,
        "difficulty": null
      },
      {
        "id": "9062237eff5e8b54b80dadfe9718886d79a70b8bc779c5f74ef308b9021de5d8",
        "domain": "numina",
        "question": "Determine the years of our century that exhibit the following property of their digits. Subtract the first digit from the second, the second from the third, and the third from the fourth, and form an arithmetic progression with these three differences.",
        "answer": "18811894",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 175684,
        "difficulty": null
      },
      {
        "id": "f954057735ec6ecc51900580a6a1d4ecd97d0702deed7bdada3dab112456f044",
        "domain": "numina",
        "question": "75. There are two identical urns. The first contains 2 black and 3 white balls, the second - 2 black and 1 white ball. First, an urn is chosen at random, and then one ball is drawn at random from it. What is the probability that a white ball will be selected?",
        "answer": "\\frac{7}{15}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32276,
        "difficulty": null
      },
      {
        "id": "8dccf88b0d7f20837b1a18391475e4b9ac88f77eeb5e2a903602f197cbe5f7d0",
        "domain": "numina",
        "question": "João has a deck of 52 cards numbered from 1 to 52. A set of three cards is called lucky if the sum of the digits on each card is the same. What is the minimum number of cards João has to take from the deck, without looking, so that among the cards he takes there are necessarily three cards that form a set of lucky cards?",
        "answer": "26",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74036,
        "difficulty": null
      },
      {
        "id": "e52144f014c3c25d51220929e99d15e0e18309d8cdc5c8e517194760319e8ba5",
        "domain": "numina",
        "question": "1614. 600 corn seeds are planted with a probability of 0.9 for each seed to germinate. Find the boundary of the absolute value of the deviation of the frequency of germinated seeds from the probability $p=0.9$, if this boundary is to be guaranteed with a probability $P=0.995$.",
        "answer": "0.034",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 30654,
        "difficulty": null
      },
      {
        "id": "52e6b5ecbf00668a394d0464badf299bd985891b81fc1c8473673089033f9e32",
        "domain": "numina",
        "question": "10. Lottery. (From 7th grade, 3 points) It so happened that the Absent-Minded Scholar had only 20 rubles left, and he needed to buy a bus ticket to go home. The ticket costs 45 rubles. Near the bus stop, instant lottery tickets are sold for exactly 10 rubles each. With a probability of \\( p = 0.1 \\), a ticket contains a win of 30 rubles, and there are no other prizes. The Scholar decided to take a risk, as he had nothing to lose. Find the probability that the Absent-Minded Scholar will be able to win enough money to buy a bus ticket.",
        "answer": "\\frac{14}{419}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10849,
        "difficulty": null
      },
      {
        "id": "8876a6112790e08264c1d73fb009b09a9627fbb0c3406589134d7c8ec61a74a0",
        "domain": "numina",
        "question": "8. On a circular track that is 660 m long, a relay race is being held, with each stage being $150 \\mathrm{~m}$. The start and finish are in the same place. What is the minimum number of stages that can be in this relay?",
        "answer": "22",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39484,
        "difficulty": null
      },
      {
        "id": "552f2270466fcab555c8d36c77a0f148baf612a0630b53207ad8336e2c04ecb3",
        "domain": "numina",
        "question": "## Task A-4.5.\n\nDetermine (if it exists) the largest natural number that cannot be represented as the sum of some, not necessarily distinct, elements of the set $\\{135,136,137, \\ldots, 144\\}$.",
        "answer": "2024",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87790,
        "difficulty": null
      },
      {
        "id": "c8935c58f9926ac43a58cea531a2cafa08a7609684d111620cc4b4d422664f3e",
        "domain": "numina",
        "question": "2. Find $x+y$, given that $x^{2}-y^{2}=10$ and $x-y=2$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152073,
        "difficulty": null
      },
      {
        "id": "dcb60b3eb5f215bb0e071ea55ada3b5284a59334dc4b0d5784154cd18822e109",
        "domain": "numina",
        "question": "Determine the largest constant $C>0$ satisfying the following property:\n\nFor any 2023-tuple $\\left(a_{1}, a_{2}, \\ldots, a_{2023}\\right)$ of strictly positive real numbers that are pairwise distinct, the following inequality holds:\n\n$$\n\\frac{a_{1}}{\\left|a_{2}-a_{3}\\right|}+\\frac{a_{2}}{\\left|a_{3}-a_{4}\\right|}+\\ldots+\\frac{a_{2023}}{\\left|a_{1}-a_{2}\\right|}>C\n$$",
        "answer": "1012",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 82110,
        "difficulty": null
      },
      {
        "id": "e62b0a1aae157bae0d1a6e2e34403e8ea47ded6eab985ea402913fc8c538c86d",
        "domain": "numina",
        "question": "2 Let convex quadrilateral $A B C D$ satisfy $A B=A D=1, \\angle A=160^{\\circ}, \\angle C=$ $100^{\\circ}$, then the range of the length of diagonal $A C$ is $\\qquad$ .",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119806,
        "difficulty": null
      },
      {
        "id": "d5ed15bf1bf492f2d33e8fc1b76e8bacb116dd6eb1a6950ff36a46088262adc3",
        "domain": "numina",
        "question": "1. On Misfortune Island, a table tennis tournament was held in a round-robin format (i.e., everyone played against everyone else once). After each match, both participants approached the Chief Referee separately and reported the result. Among the participants, there were only three types of people: knights, who always tell the truth, liars, who always lie, and politicians, who always say what is advantageous to them - each politician claimed after each match that they had won.\n\nAfter the tournament, it turned out that the Chief Referee, incognito, had attended several matches, as a result of which he was lucky enough to not only be able to determine the results of all matches but also to find out that there were exactly six politicians. What is the minimum number of matches he could have attended?\n\n(There are no draws in table tennis. The Chief Referee did not know in advance which players belonged to which type, and did not receive information in any other way except as described in the condition)",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 20948,
        "difficulty": null
      },
      {
        "id": "aa3eded0f592572c8fd913980a37647a9af4b2963d8dd2eaed94cd486356d1a3",
        "domain": "numina",
        "question": "21. (10 points) When Xiao Hong was organizing her change purse, she found that there were a total of 25 coins with denominations of 1 cent, 2 cents, and 5 cents, with a total value of 0.60 yuan. Then, the maximum number of 5-cent coins is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nNote: The instruction at the end is not part of the translation but is included to clarify the request. The actual translation is provided above.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 127149,
        "difficulty": null
      },
      {
        "id": "a0144937db035e3547dfeb0f6c837a58dde1f2d513f3997d6df3ecffbb889509",
        "domain": "numina",
        "question": "1. The price of a mathematics book in bookstore A at the beginning of January 2012 was higher than the price of the same book in bookstore B. On February 15, 2012, the price of the book in bookstore A decreased by $30\\%$, while the price of the same book in bookstore B increased by $30\\%$. Then, on March 15, 2012, the price of the book in bookstore A increased by $30\\%$, and the price of the book in bookstore B decreased by $30\\%$. The difference in the prices of the book after the changes on March 15, 2012, was 273 denars. By how much was the price of the mathematics book in bookstore A higher than the price of the same book in bookstore B at the beginning of January 2012?",
        "answer": "300",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189129,
        "difficulty": null
      },
      {
        "id": "9add5e99af303512c18d2e4175d7d8fd9d80bc0d72ffc878f413fe52b1df49b3",
        "domain": "numina",
        "question": "## Task B-1.4.\n\nDetermine the smallest natural number $n$ for which the number $2565 \\cdot n$ is a square of some natural number.",
        "answer": "285",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87254,
        "difficulty": null
      },
      {
        "id": "408224bcc3ffec6d5ce5ff9db2f6ad6ad58c9395cbc88cd481753889ec9af97a",
        "domain": "numina",
        "question": "A car and a minivan drive from Alphaville to Betatown. The car travels at a constant speed of $40 \\mathrm{~km} / \\mathrm{h}$ and the minivan travels at a constant speed of $50 \\mathrm{~km} / \\mathrm{h}$. The minivan passes the car 10 minutes before the car arrives at Betatown. How many minutes pass between the time at which the minivan arrives in Betatown and the time at which the car arrives in Betatown?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194309,
        "difficulty": null
      },
      {
        "id": "fe90e0ece83db6572e555fac73d18c563b1f90f8045b9992e14554109c58b64e",
        "domain": "numina",
        "question": "Example 1 Calculate $\\left|\\frac{(\\sqrt{3}+\\sqrt{2} i)(\\sqrt{5}+\\sqrt{2} i)(\\sqrt{5}+\\sqrt{3} i)}{(\\sqrt{2}-\\sqrt{3} i)(\\sqrt{2}-\\sqrt{5} i)}\\right|$\n(1983 Shanghai High School Competition)",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110541,
        "difficulty": null
      },
      {
        "id": "d64454b8f3d38ff1cbd89a4518042c531a74dbcda69acd2fae512d528a23adec",
        "domain": "numina",
        "question": "3. On the surface of the cube $A B C D E F G H$, a broken line composed of four identical segments is constructed in the faces $A B F E, B C G F, C D H G$ and $G H E F$, which starts from vertex $A$ and ends at vertex $E$. Determine in what ratio this broken line divides the edge $C G$.",
        "answer": "\\frac{1}{2}(\\sqrt{5}-1)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75557,
        "difficulty": null
      },
      {
        "id": "dc97e6851763673d55e41cae419e008fb00a67014296594bebb161fd3aabe325",
        "domain": "numina",
        "question": "8. (10 points) The difference between the minimum value of the sum of the squares of ten different odd numbers and the remainder when this minimum value is divided by 4 is $\\qquad$\n(Note: The product of the same two natural numbers is called the square of this natural number, such as $1 \\times 1=1^{2}, 2 \\times 2=2^{2}, 3 \\times 3$ $=3^{3}$, and so on)",
        "answer": "1328",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 135177,
        "difficulty": null
      },
      {
        "id": "4b4cb0447804a53a21620c33bc354c8ca9af434fda6555fd04685a65a938ec98",
        "domain": "numina",
        "question": "(7) Given positive integers $x, y, z$ satisfy the condition $x y z=(14-x)(14-y)(14-$\n$z)$, and $x+y+z<28$, then the maximum value of $x^{2}+y^{2}+z^{2}$ is $\\qquad$.",
        "answer": "219",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144768,
        "difficulty": null
      },
      {
        "id": "a32a63a7a53ac11e061055335afdaab1ff44b01a44e7f0bff0c971c93e03b7bd",
        "domain": "numina",
        "question": "9. Bear Da painted the outer surface of a cubic wooden block blue, then divided it into many smaller cubic wooden blocks of the same size, among which 156 smaller cubic wooden blocks have two blue faces. Therefore, the number of smaller cubic wooden blocks with no blue faces is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "2197",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133491,
        "difficulty": null
      },
      {
        "id": "dbd529ce0489353020607657af64e358f3119801366daa582fe89d526a7cadd4",
        "domain": "numina",
        "question": "$2 \\cdot 34$ Let $S$ be a subset of $\\{1,2, \\cdots, 1989\\}$, and any two numbers in $S$ do not have a difference of 4 or 7. How many elements can $S$ have at most?",
        "answer": "905",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119491,
        "difficulty": null
      },
      {
        "id": "f8ef27fc089300713c12222644e9e12cd462411f47d242566194df14d5407175",
        "domain": "numina",
        "question": "Problem 8.5. In a convex quadrilateral $A B C D$, the bisectors of angles $A$ and $C$ are parallel, and the bisectors of angles $B$ and $D$ intersect at an angle of $46^{\\circ}$, as shown in the figure. How many degrees does the acute angle between the bisectors of angles $A$ and $B$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_191a5994d6b917032c66g-03.jpg?height=490&width=616&top_left_y=94&top_left_x=419)",
        "answer": "67",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13947,
        "difficulty": null
      },
      {
        "id": "f192860ded804f4a8a069223cd411b548ce045f777b0ef6524309f3565e2088f",
        "domain": "numina",
        "question": "3. The two diagonals of a quadrilateral are perpendicular. The lengths of the diagonals are 14 and 30 . What is the area of the quadrilateral?",
        "answer": "210",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156491,
        "difficulty": null
      },
      {
        "id": "45ade1c04737729ba1c31d4309eb0de582f06e54e1883ad32e88546bb3d1db7d",
        "domain": "numina",
        "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 72, 28, and 28, and the angles at their vertices are $-\\frac{\\pi}{3}$, $\\frac{2 \\pi}{3}$, and $\\frac{2 \\pi}{3}$ respectively (the angle at the vertex of a cone is the angle between its generators in the axial section). A sphere is suspended above the table, touching all the cones. It turns out that the center of the sphere is equidistant from all points of contact with the cones. Find the radius of the sphere.",
        "answer": "\\frac{\\sqrt{3}+1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7664,
        "difficulty": null
      },
      {
        "id": "b418994fbf7ad69dbf1dcc1a6e4d6b3fc86e2288b87fdcf3a68b7a79295824ed",
        "domain": "numina",
        "question": "Question 179, As shown in the figure, in the Cartesian coordinate system, the ellipse $\\Gamma: \\frac{x^{2}}{2}+y^{2}=1$ has left and right foci $F_{1}$ and $F_{2}$, respectively. Let $P$ be a point on $\\Gamma$ in the first quadrant, and the extensions of $P F_{1}$ and $P F_{2}$ intersect $\\Gamma$ at points $Q_{1}$ and $Q_{2}$, respectively. Let $r_{1}$ and $r_{2}$ be the inradii of $\\triangle P F_{1} Q_{2}$ and $\\triangle P F_{2} Q_{1}$, respectively. Find the maximum value of $r_{1}-r_{2}$.",
        "answer": "\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 117955,
        "difficulty": null
      },
      {
        "id": "354f4581b2d72c263db2c773ab594457fb8bf9269e6b909295ee1ccd8497d1b9",
        "domain": "numina",
        "question": "5 married couples gather at a party. As they come in and greet each other, various people exchange handshakes - but, of course, people never shake hands with themselves or with their own respective spouses. At the end of the party, one woman goes around asking people how many hands they shook, and she gets nine different answers. How many hands did she herself shake?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 90470,
        "difficulty": null
      },
      {
        "id": "d9282ab391c74839440bbd41caa65d8f5c57e51f622340fd61941185a4924c1b",
        "domain": "numina",
        "question": "22. Find the value of $\\frac{\\tan 40^{\\circ} \\tan 60^{\\circ} \\tan 80^{\\circ}}{\\tan 40^{\\circ}+\\tan 60^{\\circ}+\\tan 80^{\\circ}}$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160387,
        "difficulty": null
      },
      {
        "id": "a48f8a5353b3910dc9c89f1e8e90ce11b245b4376d6dc9a637894d02de551224",
        "domain": "numina",
        "question": "Fomin D.\n\nOn the board, the numbers $1, 1/2, 1/3, \\ldots, 1/100$ are written. We choose two arbitrary numbers $a$ and $b$ from those written on the board, erase them, and write the number\n\n$a + b + ab$. We perform this operation 99 times until only one number remains. What is this number? Find it and prove that it does not depend on the sequence of number selections.",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69954,
        "difficulty": null
      },
      {
        "id": "e3ec74383eb20a7939d179cb36d64661ea2ac7520f2a337a8d782fd975e4e3ae",
        "domain": "numina",
        "question": "3.2. Znayka cut out a semicircle from paper. Neznaika marked a point $C$ on the diameter $A B$ of this semicircle and cut off two semicircles from Znayka's semicircle with diameters $A C$ and $C B$. Find the area of the remaining figure if the length of the part of the chord lying inside it, passing through point $C$ perpendicular to $A B$, is 8. If necessary, round your answer to two decimal places.",
        "answer": "50.27",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5816,
        "difficulty": null
      },
      {
        "id": "568795de4b3779836044e5865b56258fcb87d9007b44d83b4acffeb06834ad5c",
        "domain": "numina",
        "question": "2. At the Olympiad, for each solved problem, one could receive 3, 8, or 10 points. Vasya scored 45 points. What is the smallest number of problems he could have solved? (It is necessary to explain why he could not have solved fewer problems.)",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15702,
        "difficulty": null
      },
      {
        "id": "99202d7cc0b8f0f11344a8862c857c0848dc3317ee8176ed97b1903fdbb71e3b",
        "domain": "numina",
        "question": "Point $D$ lies on side $CB$ of the right triangle $ABC\\left(\\angle C=90^{\\circ}\\right)$, and $AB=5, \\angle ADC=\\arccos$ $\\frac{1}{\\sqrt{10}}, DB=\\frac{4 \\sqrt{10}}{3}$. Find the area of triangle $ABC$.",
        "answer": "\\frac{15}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59825,
        "difficulty": null
      },
      {
        "id": "b381f96c74534e6cf20298085fe3ffcf76fb4a044daed0472929549e7f13d41e",
        "domain": "numina",
        "question": "\\section*{Problem 4 - 051044}\n\nCalculate the difference \\(D\\) between the sum of the squares of all even natural numbers \\(\\leq 100\\) and the sum of the squares of all odd natural numbers \\(<100\\)!",
        "answer": "5050",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 184944,
        "difficulty": null
      },
      {
        "id": "dd341eaa40904cca6fe86d705b58342c47ab27fe1a1ce1115a791afdaff36590",
        "domain": "numina",
        "question": "G8.2 From 1 to 121 , there are $b$ numbers which are not divisible by 5 nor 7 . Find $b$.",
        "answer": "83",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152723,
        "difficulty": null
      },
      {
        "id": "bceff789e87e7cebe8f91129c61018bf5ba0eecea24b38b17459ef745e281c51",
        "domain": "numina",
        "question": "14. 2011 cards are arranged in a row on a table. One of the numbers ' 1 ', ' 2 ' and ' 3 ' is printed on each card. It is found that there is at least one card between any two cards labelled ' 1 ', at least two cards between any two cards labelled ' 2 ', and at least three cards between any two cards labelled ' 3 '. If the smallest and greatest possible numbers of cards labelled ' 3 ' are $m$ and $M$ respectively, find the value of $m+M$.\n2011 cards are placed in a row on a table, each card has one of the numbers '1', '2', or '3' written on it. It is known that there is at least one card between any two cards labelled '1', at least two cards between any two cards labelled '2', and at least three cards between any two cards labelled '3'. If the smallest and greatest possible numbers of cards labelled '3' are $m$ and $M$ respectively, find the value of $m+M$.",
        "answer": "1005",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153625,
        "difficulty": null
      },
      {
        "id": "7cd8d3c81327ebe10ac97c5b45a5d3cef56fceeb35211894b399c5584c6eae94",
        "domain": "numina",
        "question": "2. First method: Since $x_{1}^{3}-2015 x_{1}+2016=0$ and $x_{2}^{3}-2015 x_{2}+2016=0$, then $x_{1}^{3}-x_{2}^{3}=2015 \\cdot\\left(x_{1}-x_{2}\\right)$. Therefore, $x_{1}^{2}+x_{1} x_{2}+x_{2}^{2}=2015$.\n\nSecond method: By Vieta's theorem (but then it needs to be justified that there are three distinct roots): $x_{1}+x_{2}+x_{3}=0, x_{1} x_{2} x_{3}=-2016$. Therefore,\n\n$$\n\\begin{aligned}\nx_{1}^{2}+x_{1} x_{2}+x_{2}^{2}=\\left(x_{1}+x_{2}\\right)^{2}-x_{1} x_{2}=\\left(-x_{3}\\right)^{2}+\\frac{2016}{x_{3}} & \\\\\n=\\frac{x_{3}^{3}+2016}{x_{3}} & =\\frac{2015 x_{3}}{x_{3}}=2015\n\\end{aligned}\n$$",
        "answer": "2015",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5174,
        "difficulty": null
      },
      {
        "id": "9c02001a35b06860872893ed59ac0fc8fd9166a97017e63a637651bb26b7721f",
        "domain": "numina",
        "question": "## Task 2 - V10922\n\nAccording to our seven-year plan, the industrial production of the German Democratic Republic will increase significantly. The total industrial production will grow by 88% from 1958 to 1965. The production of means of production (i.e., raw materials, machinery, equipment for industry, agriculture, and transportation, etc.) will increase by 95%, while the production of consumer goods (i.e., goods intended for the needs of the population) will increase by 77%.\n\nWhat percentage of the total industrial production did the production of means of production account for in 1958? What percentage will it account for in 1965?",
        "answer": "61.1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184354,
        "difficulty": null
      },
      {
        "id": "8744a21cbffd322ee3029b905a613da57c6d6d37782da9bfe12adbf64521e14e",
        "domain": "numina",
        "question": "To punish his class for indiscipline, Professor Zerus decided to deduct from each student's monthly grade a percentage equal to the test score, that is, whoever scored 60 will have a 60% deduction from their grade, whoever scored 20 will have a 20% deduction, and so on. The maximum monthly grade is 100.\n\n(a) Who will end up with the highest grade?\n\n(b) And the lowest?\n(c) Students who scored well complained that they will end up with the same grade as those who scored poorly. Do they have a point?",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 74790,
        "difficulty": null
      },
      {
        "id": "88805bc15c8e6af81894c003a1fc7d79490094766acdafa80c09410533376b0a",
        "domain": "numina",
        "question": "I2.3 If $c$ is the minimum value of the quadratic function $y=x^{2}+4 x+b$, find the value of $c$.",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152822,
        "difficulty": null
      },
      {
        "id": "13df2c593736a6efeb520d2626270de86b89f92a2538b198ba2630768fa3d9da",
        "domain": "numina",
        "question": "3. Given an isosceles triangle \\(ABC\\) with base \\(BC\\). On the extension of side \\(AC\\) beyond point \\(C\\), point \\(K\\) is marked, and a circle is inscribed in triangle \\(ABK\\) with center at point \\(I\\). A circle passing through points \\(B\\) and \\(I\\) is tangent to line \\(AB\\) at point \\(B\\). This circle intersects segment \\(BK\\) again at point \\(L\\). Find the angle between lines \\(IK\\) and \\(CL\\). Answer: \\(90^{\\circ}\\).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c43b1f7fb831d10d42ecg-02.jpg?height=496&width=820&top_left_y=483&top_left_x=638)",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7738,
        "difficulty": null
      },
      {
        "id": "24dd06e0d7e618eee500d05c64cd98a6fae41c3c9dc72ed5030562370495952c",
        "domain": "numina",
        "question": "## 39. Square Plot\n\nMathieu has a square plot of land. Surrounding this plot is an alley of constant width with an area of $464 \\mathrm{~m}^{2}$. Walking around his plot, Mathieu noticed that the difference in length between the outer and inner edges of the alley is 32 m.\n\nWhat is the total area of Mathieu's plot, including the alley?",
        "answer": "1089",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 42218,
        "difficulty": null
      },
      {
        "id": "b1053d81de79ced248beb17fc99650fe95b9694603c0b64533cd6278f1a5dc22",
        "domain": "numina",
        "question": "10. Given that point $P$ lies on the hyperbola $\\frac{x^{2}}{16}-\\frac{y^{2}}{9}=1$, and the distance from $P$ to the right directrix of this hyperbola is exactly the arithmetic mean of the distances from $P$ to the two foci of this hyperbola. Then, the x-coordinate of $P$ is $\\qquad$ .",
        "answer": "-\\frac{64}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127586,
        "difficulty": null
      },
      {
        "id": "6e459e60e9accc6fce0469f10e9082b8e223b000e9148fba2962940406338fda",
        "domain": "numina",
        "question": "14. A rectangular prism with integer edge lengths is painted red on all its surfaces, and then it is cut into small cubes with edge lengths of 1. Among them, there are 40 small cubes with two red faces, and 66 small cubes with one red face. What is the volume of this rectangular prism?",
        "answer": "150",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 139622,
        "difficulty": null
      },
      {
        "id": "25b2f566cb40e3328927d76517b5ea63eb3eba27ef12acc19010951c83b0f13b",
        "domain": "numina",
        "question": "Problem 1. Find the least positive integer $n$ with the following property: if $n$ distinct sums of the form $x_{p}+x_{q}+x_{r}, 1 \\leq p<q<r \\leq 5$, equal 0 , then $x_{1}=x_{2}=x_{3}=x_{4}=x_{5}=0$.\n\nSava Grozdev, Svetlozar Doychev",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 177495,
        "difficulty": null
      },
      {
        "id": "51fa53e2b3ffe541cb1e420f925e478d6d4c91811c9030c94509122a4a97a59c",
        "domain": "numina",
        "question": "13 The function $f(x)=2 \\sin \\omega x(\\omega>0)$ is monotonically increasing on $\\left[0, \\frac{\\pi}{4}\\right]$, and the maximum value on this interval is $\\sqrt{3}$. Then, $\\omega=$ $\\qquad$ .",
        "answer": "\\frac{4}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120523,
        "difficulty": null
      },
      {
        "id": "f7bfa8158b39942718c74c98e509059bb9eaa71fddb94d1c09801fa459d745e9",
        "domain": "numina",
        "question": "Mr. Krbec with his cat Kokeš were selling tickets at Kulikov Castle. On Saturday, they sold 210 children's tickets for 25 groschen each and also some adult tickets for 50 groschen each. In total, they earned 5950 groschen that day.\n\nHow many adult tickets did they sell?\n\n(M. Krejčová)\n\nHint. How much did they earn from the adult tickets?",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76686,
        "difficulty": null
      },
      {
        "id": "ab7bef8a7ed7ee3ef932d6dffc699a140676849647be7a107c27d551ff4afc04",
        "domain": "numina",
        "question": "43.3. In a circle, mutually perpendicular diameters $A B$ and $C D$ are drawn, and an arbitrary point $M$ is taken on the arc $A C$. Find $M B + M D$, if $M A + M C = a$.\n\n$$\n\\text { (9-10 grades) }\n$$",
        "answer": "(1+\\sqrt{2})",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 45060,
        "difficulty": null
      },
      {
        "id": "60137279713bf47e3a70f5ebab061aba2afd931375a607c79f89dac672d9483f",
        "domain": "numina",
        "question": "19. Let $O$ be the origin, $A_{1}, A_{2}, A_{3}, \\ldots$ be points on the curve $y=\\sqrt{x}$ and $B_{1}, B_{2}, B_{3}, \\ldots$ be points on the positive $x$-axis such that the triangles $O B_{1} A_{1}, B_{1} B_{2} A_{2}, B_{2} B_{3} A_{3}, \\ldots$ are all equilateral, with side lengths $l_{1}, l_{2}, l_{3}, \\ldots$ respectively. Find the value of $l_{1}+l_{2}+l_{3}+\\cdots+l_{2005}$.\n(2 marks)\nLet $O$ be the origin, $A_{1}, A_{2}, A_{3}, \\ldots$ be points on the curve $y=\\sqrt{x}$ and $B_{1}, B_{2}, B_{3}, \\ldots$ be points on the positive $x$-axis such that the triangles $O B_{1} A_{1}, B_{1} B_{2} A_{2}, B_{2} B_{3} A_{3}, \\ldots$ are all equilateral, with side lengths $l_{1}, l_{2}, l_{3}, \\ldots$ respectively. Find the value of $l_{1}+l_{2}+l_{3}+\\cdots+l_{2005}$.",
        "answer": "\\frac{4022030}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153506,
        "difficulty": null
      },
      {
        "id": "ce5fc4f849f58ec70868dc1c4632ab9cfaf22b1e3ae7aa525c2545bcb30e405f",
        "domain": "numina",
        "question": "8. If positive integers $a, b, c$ satisfy $2017 \\geqslant 10 a \\geqslant 100 b \\geqslant 1000 c$, then the number of tuples $(a, b, c)$ is\n$\\qquad$ .",
        "answer": "574",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118989,
        "difficulty": null
      },
      {
        "id": "ca52f6ad2117a73d80fb040278bff7c8d28765f4f7bad2e6e5dccd0a69aa8455",
        "domain": "numina",
        "question": "Three concentric circles have radii of 1, 2, and 3 units, respectively. A point is chosen on each of the three circles such that they form the vertices of an equilateral triangle. What can be the side length of this equilateral triangle?",
        "answer": "\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170219,
        "difficulty": null
      },
      {
        "id": "2e46697364b8a246ab4bd16f83fe49a337c0625589132dbffb84f4a10506639d",
        "domain": "numina",
        "question": "8. (10 points) As shown in the figure, the area of trapezoid $ABCD$ is 320, the area of parallelogram $CDEF$ is 240, and the area of triangle $OCD$ is $\\qquad$\n\n保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 132963,
        "difficulty": null
      },
      {
        "id": "80c10df6ec51d02b433a09dc4c32acfd1c5483e52f9dea20345c76753458048c",
        "domain": "numina",
        "question": "10.5. On side $AB$ of triangle $ABC$, point $K$ is marked, and on side $AC$, point $M$ is marked. Segments $BM$ and $CK$ intersect at point $P$. It turns out that angles $APB$, $BPC$, and $CPA$ are each $120^{\\circ}$, and the area of quadrilateral $AKPM$ is equal to the area of triangle $BPC$. Find angle $BAC$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15685,
        "difficulty": null
      },
      {
        "id": "971df300c40601a3b0e708baa837de270f7635016c35e09443f5091d997bbd52",
        "domain": "numina",
        "question": "51. Given $\\triangle A B C$ with three interior angles $\\angle A, \\angle B, \\angle C$ and $\\alpha=\\angle A+\\angle B, \\beta=\\angle C+\\angle A, \\gamma$ $=\\angle C+\\angle B$, the maximum number of acute angles among $\\alpha, \\beta, \\gamma$ is . $\\qquad$",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127485,
        "difficulty": null
      },
      {
        "id": "37d15ad98222eb3a9956ff1786516917233c0d891af583024381aaf4fc568440",
        "domain": "numina",
        "question": "10. In 2021, the difference between the digits of any two adjacent positions does not exceed 2 (subtracting the smaller number from the larger one). If all four-digit numbers where the difference between the digits of any two adjacent positions does not exceed 2 are arranged in ascending order, then 2021 is the $\\qquad$th number in this sequence.",
        "answer": "84",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 97216,
        "difficulty": null
      },
      {
        "id": "2b01b3e901e9ca3ad440b872de12da43df4ba2ac04319380f4064dcc28b82e58",
        "domain": "numina",
        "question": "4. If the inequality\n$$\n\\left|a x^{2}+b x+a\\right| \\leqslant x\n$$\n\nholds for all $x \\in[1,2]$, then the maximum value of $3 a+b$ is\n$\\qquad$ .",
        "answer": "3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 111977,
        "difficulty": null
      },
      {
        "id": "6c8e3c47aae9e719b0bf66d39324bbe128da82481c7fd7ab29c110a3c07bb4bb",
        "domain": "numina",
        "question": "176 Given the parabola $y^{2}=x, A B$ is a chord passing through the focus $F$, $A$ and $B$ are in the $\\mathrm{I}$ and $\\mathrm{N}$ quadrants respectively, if $A F-F B=1$, then the slope of $A B$ $=$ $\\qquad$",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93752,
        "difficulty": null
      },
      {
        "id": "dbf745373af266ed00c88efaf95595acc7ed1ff2eaf7b6891f8d06bc9be4af2b",
        "domain": "numina",
        "question": "5. In a hat, there are 11 slips of paper with numbers $0,1,2,3,4,5,6,7,8,9,10$ (one with each number). Each of the eleven students takes one slip. Then they announce in alphabetical order the sentence \"so far, at least $k$ students have told a lie,\" where $k$ is the number on the drawn slip. How many of these 11 statements can be true at most?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 90044,
        "difficulty": null
      },
      {
        "id": "929ad6660ce9ffdca4e0f3b25a935068853b57347cd3bd817fbb94a81bbc9f92",
        "domain": "numina",
        "question": "## Task 3 - V00803\n\nI drop a ball. It bounces up to $\\frac{2}{3}$ of its falling height. It falls again and bounces the second time $\\frac{5}{8}$ of the first bounce height.\n\nCalculate from what height I dropped the ball if it bounced $45 \\mathrm{~cm}$ less high the second time than the first time!",
        "answer": "180",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183891,
        "difficulty": null
      },
      {
        "id": "700608239195ec4a445e72503d28614b5d6f64dddb8cb64ed875c48da1df34f7",
        "domain": "numina",
        "question": "1. Find the minimum value of the expression $\\frac{25 x^{2} \\sin ^{2} x+16}{x \\sin x}$ for $0<x<\\pi$.",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17610,
        "difficulty": null
      },
      {
        "id": "24a44e6b43bdc52d3017b92ab507df91616aaaae1f38ceb742c58496ef8305fd",
        "domain": "numina",
        "question": "39. How far is it to school?\n\nAll the time while Alice and the Griffin were solving the previous problem, the Tortoise Quasi was weeping inconsolably.\n\n- Can you tell me what is sad about this problem? - the Griffin barked at him angrily. - After all, the passenger caught up with the train. Or did I misunderstand something?\n- Everything is correct, - agreed the Tortoise Quasi, - but we don't know what happened afterward! After all, judging by everything, the train could have derailed!\n- No, just think about it! - Alice got angry. - First, they come up with all sorts of nonsense, and then they make a sad story out of everything!\n\nThe Tortoise Quasi did not answer, only grabbing his head with his front paws.\n\n- Alright, - he said finally. - I will tell you a sad story. One morning, a boy had to go to school...\n- That's sad! - agreed the Griffin.\n- No, no! That's not sad, - objected the Tortoise Quasi. - The saddest part is still to come.\n\nAlice and the Griffin listened to the whole story attentively but found nothing sad in it.\n\n- So, - continued the Tortoise Quasi, - the father said to the boy: \"Hurry up, or you'll be late for school!\" - \"I won't be late, Dad! - the boy replied. - I have everything calculated precisely. If I walk at a speed of four miles per hour, I will be five minutes late for the start of classes, but if I walk at a speed of five miles per hour, I will arrive ten minutes before the start of classes.\"\n\nHow far is it to school?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36518,
        "difficulty": null
      },
      {
        "id": "09632a26f0cd2aa6a4b4a164e7a1fd8555e0f95f810cb208773cc6802b8a77e8",
        "domain": "numina",
        "question": "One, (25 points) Given the family of curves $2(2 \\sin \\theta-\\cos \\theta+3) x^{2}-(8 \\sin \\theta+\\cos \\theta+1) y=0, \\theta$ being the parameter, find the maximum length of the chord intercepted by the line $y=2 x$ on this curve.\n\n---\n\nThe translation retains the original text's line breaks and formatting.",
        "answer": "8\\sqrt{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92807,
        "difficulty": null
      },
      {
        "id": "76077ab25e66b072d2adc6f7bec0f91f03a674c0a8382bdb2fa0d7c89a32adaa",
        "domain": "numina",
        "question": "Third question: For all real numbers $x_{1}, x_{2}, \\ldots, x_{60} \\in[-1,1]$, find the maximum value of $\\sum_{i=1}^{60} x_{i}^{2}\\left(x_{i+1}-x_{i-1}\\right)$, where $x_{0}=x_{60}, x_{61}=x_{1}$.",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112775,
        "difficulty": null
      },
      {
        "id": "753f7d74c0d73fce9b849b6e5f58450013d33cf00f32075b519372bf206953a5",
        "domain": "numina",
        "question": "2. For the number $a$, the equality $a+\\frac{1}{a}=1$ is satisfied. Calculate the value of\n\n$$\na^{5}+\\frac{1}{a^{5}}\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190788,
        "difficulty": null
      },
      {
        "id": "9157db8caed1f30a23f550c41e29b32ed5540d8492a5ef501e8b45a4803797ff",
        "domain": "numina",
        "question": "4. Wu Yu wrote three letters and three envelopes, and he needs to put each letter into the corresponding envelope, with only one letter per envelope. The number of all possible scenarios where at least one of the three letters is placed incorrectly is $\\qquad$ kinds.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 133935,
        "difficulty": null
      },
      {
        "id": "e64ad34ec9850a3f20f2d82822886ca900591d93d2ca86d2447e026f3b210639",
        "domain": "numina",
        "question": "6 books are aligned in a library. How many ways are there to arrange them.",
        "answer": "720",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 81732,
        "difficulty": null
      },
      {
        "id": "470df238e2c4bd140538b2da6af8f2d00f4c4762fac43860908b6d511b039c8a",
        "domain": "numina",
        "question": "## Task 25/62\n\nAn apprentice is supposed to count 1000 balls with a diameter of $1 \\mathrm{~cm}$ in a ball bearing factory. To speed up this task, he takes a container with inner dimensions of $10 \\mathrm{~cm} \\times 10 \\mathrm{~cm} \\times 10$ $\\mathrm{cm}$; he lays the first layer neatly and then continues to fill it.\n\nIn the end, he finds that, contrary to his expectations, the inner space of the container is not completely filled. He therefore counts the balls. Surprisingly, there are more than a thousand.\n\nHow many were there, and how many centimeters were missing from the top layer of balls to the edge of the container?",
        "answer": "1186",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 182017,
        "difficulty": null
      },
      {
        "id": "3bc5cdaee7ae40fc8f89cc4ce54f2f9bce37a7f52327bf4fd70fbe37be22b16e",
        "domain": "numina",
        "question": "46. A U-shaped number is a special type of three-digit number where the units digit and the hundreds digit are equal and larger than the tens digit, such as 818 is a $\\cup$ shaped number. There are $\\qquad$ U-shaped numbers.",
        "answer": "45",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 110042,
        "difficulty": null
      },
      {
        "id": "9958d0add4295348e64c57c68a293bf4a9e3680f63b7d0e4f1e4464882f6fa3d",
        "domain": "numina",
        "question": "15. Given the ellipse $E: \\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$, its minor axis length is 4, and the eccentricity is $e_{1}$. The hyperbola $\\frac{x^{2}}{m}-\\frac{y^{2}}{n}=1(m>0$, $n>0)$ has asymptotes $y= \\pm x$, and the eccentricity is $e_{2}$, and $e_{1} \\cdot e_{2}=1$.\n(1) Find the equation of the ellipse $E$;\n(2) Let the right focus of the ellipse $E$ be $F$, and draw a line through the point $G(4,0)$ with a non-zero slope intersecting the ellipse $E$ at points $M, N$. Let the slopes of the lines $F M$ and $F N$ be $k_{1}, k_{2}$, respectively. Determine whether $k_{1}+k_{2}$ is a constant value. If it is a constant value, find the value; if not, explain the reason.",
        "answer": "0",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147163,
        "difficulty": null
      },
      {
        "id": "f6d80e984e0d94166a5a474bb5b48a888db4e6c4f4e94ec00915b95e6392557c",
        "domain": "numina",
        "question": "Problem 6. (30 points) Ten balls of the same radius are arranged in the form of a triangular pyramid such that each ball touches at least three others. Find the radius of the sphere in which the pyramid of balls is inscribed, if the radius of the sphere inscribed in the center of the pyramid of balls, touching six identical balls, is $\\sqrt{2}-1$.\n\n#",
        "answer": "\\sqrt{6}+1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7180,
        "difficulty": null
      },
      {
        "id": "68f97edc32ada08f863b005773f431f718e4ea0c82f3f2668ad294314e6756a9",
        "domain": "numina",
        "question": "Given\n$$\nf(x y)=f(x)+f(y)\n$$\n\nand $f(2)=1$. Find the value of $f\\left(\\frac{1}{64}\\right)$.",
        "answer": "-6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123590,
        "difficulty": null
      },
      {
        "id": "121d5df462cba30675c165f88dcada38749e8ea123da90e489cdd8f93d700ed1",
        "domain": "numina",
        "question": "7.5. The following numbers are written on the wall: $1,3,4,6,8,9,11,12,16$. Four of these numbers were written by Vova, four numbers were written by Dima, and one number was simply the house number of the local police officer. The police officer found out that the sum of the numbers written by Vova is three times the sum of the numbers written by Dima. Calculate the house number.",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 15240,
        "difficulty": null
      },
      {
        "id": "f57ddc2fd790a8eefc91a32c33d7579405870a8b396c9c6163449cc59c99a443",
        "domain": "numina",
        "question": "Task B-3.3. Determine the fundamental period of the function $f(x)=8 \\sin ^{2} x \\cos ^{2} x-2 \\cos ^{2} 2 x$.",
        "answer": "\\frac{\\pi}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88894,
        "difficulty": null
      },
      {
        "id": "65bc5bcd103084705f08771cfb7e578af3fd123b5d192b194eba7b5b6eeada84",
        "domain": "numina",
        "question": "Shapavaooov A.B.\n\na) On each of the fields of the top and bottom horizontal rows of an $8 \\times 8$ chessboard, there is a token: white ones at the bottom, black ones at the top. In one move, it is allowed to move any token to an adjacent free cell vertically or horizontally. What is the minimum number of moves required to achieve a configuration where all black tokens are at the bottom and all white tokens are at the top?\n\nb) The same question for a $7 \\times 7$ board.",
        "answer": "120",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 70539,
        "difficulty": null
      },
      {
        "id": "dac4b30d61b23a891496baf5a3163d0918c7f5387bff6ce8d0a4b04dd15dcafc",
        "domain": "numina",
        "question": "On a quiz, every question is solved by exactly four students, every pair of questions is solved by exactly one student, and none of the students solved all of the questions. Find the maximum possible number of questions on the quiz.",
        "answer": "13",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 90801,
        "difficulty": null
      },
      {
        "id": "b07a5e78c4433a42bc57d416eef8cc24c32301956c0d8be6f5181c9d54d19059",
        "domain": "numina",
        "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $12$, and $\\cos \\alpha = \\frac{3}{4}$?",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19465,
        "difficulty": null
      },
      {
        "id": "24fec2005513b77d5270cdfda7f3ab0a9ab85d4b148d3ccea5d56ea26ef96f90",
        "domain": "numina",
        "question": "8. (5 points) As shown in the figure, in the right triangle $\\triangle ABC$, the hypotenuse $AB=10, BC=5, \\angle ABC=60^{\\circ}$. With point $B$ as the center, $\\triangle ABC$ is rotated clockwise by $120^{\\circ}$, and points $A$ and $C$ reach points $E$ and $D$, respectively. The area swept by side $AC$ (i.e., the shaded area in the figure) is $\\qquad$. (Take $\\pi$ as 3)",
        "answer": "75",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 111016,
        "difficulty": null
      },
      {
        "id": "60294e988b66df18c55e6c99fb682f045e756246d6178b5905beef4d568ced17",
        "domain": "numina",
        "question": "8. For a regular 2017-gon $A_{1} A_{2} \\cdots A_{2017}$ inscribed in the unit circle $O$, if any two distinct vertices $A_{i}, A_{j}$ are chosen, then the probability that $\\overrightarrow{O A_{i}} \\cdot \\overrightarrow{O A_{j}}>\\frac{1}{2}$ is $\\qquad$.",
        "answer": "\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 138554,
        "difficulty": null
      },
      {
        "id": "bfa258c73a05a31f757daa0343425cf22d129d434ad7f3c0d6bc6ebf3d610eda",
        "domain": "numina",
        "question": "02.2. In two bowls there are in total $N$ balls, numbered from 1 to $N$. One ball is moved from one of the bowls to the other. The average of the numbers in the bowls is increased in both of the bowls by the same amount, $x$. Determine the largest possible value of $x$.",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 420,
        "difficulty": null
      },
      {
        "id": "2cd5d5b986dc7780bce44d2ea5cea53d8e9a3bb50096c0d347af38046ac604aa",
        "domain": "numina",
        "question": "Problem 3. Let $\\triangle A B C$ with $m(\\measuredangle A)=90^{\\circ}$. Construct the square $B D E C$ in the half-plane delimited by $B C$ that does not contain $A$. The angle bisector of $\\angle A$ intersects the sides $[B C]$ and $[D E]$ at $F$ and $G$ respectively. If $|A B|=16 \\mathrm{~cm}$ and $|A C|=4 \\mathrm{~cm}$, calculate the area of the quadrilateral $B D G F$.",
        "answer": "136",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 179955,
        "difficulty": null
      },
      {
        "id": "3c141303f7b16e5c33b62ad3d69e8ac8c93ca2b4b37b2ba8a80efa2e801f8c83",
        "domain": "numina",
        "question": "As shown in the figure, in an $8 \\times 8$ square grid, there are two bugs at points $A$ and $B$ (the bug at point $A$ is called bug $a$, and the bug at point $B$ is called bug $b$). Each bug moves 1 grid (choosing one of the four directions: up, down, left, or right). If bug $b$ moves two times, and bug $a$ moves three times, the number of ways in which the distance from bug $b$ to point $A$ is less than or equal to the distance from bug $a$ to point $A$ is $\\qquad$ ways.",
        "answer": "748",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 145387,
        "difficulty": null
      },
      {
        "id": "65a5e7e0311588426ea2b1cc0ff2800679bd1bf0f6e5ef6e1dbce9aa2615db41",
        "domain": "numina",
        "question": "5. In the rectangular prism $A B C D-E F G H$ shown in the figure, let $P$ be the center of rectangle $E F G H$, and line segment $A P$ intersects plane $B D E$ at point $Q$. If $A B=3, A D=2, A E=$ 1, then $P Q=$ $\\qquad$.",
        "answer": "\\frac{\\sqrt{17}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 125476,
        "difficulty": null
      },
      {
        "id": "880cf81df0aed46f6982e62ca56f28f061cfe4fa03f551a84ea359a9c9d3edfc",
        "domain": "numina",
        "question": "60. A technology company produced 17 units and 15 units of the same model of testing equipment in locations A and B, respectively, all to be transported to venues $A$ and $B$ of a major sports event, with 18 units going to venue $A$ and 14 units to venue $B$. The transportation costs to venues $A$ and $B$ are as follows:\n（1）Assume that $x$ units of equipment are transported from location A to venue $A$. Please fill in the table below and find the function relationship between the total transportation cost $y$ (yuan) and $x$ (units);\n（2）When $x$ is $\\qquad$, the total transportation cost is minimized, with the minimum value being $\\qquad$",
        "answer": "19900",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113179,
        "difficulty": null
      },
      {
        "id": "9521ecba341301e73e9f1a23913438c76557292a2070c19a90b35423081538f9",
        "domain": "numina",
        "question": "2. The car traveled half of the distance at a speed of 60 km/h, then one third of the remaining distance at a speed of 120 km/h, and the remaining distance at a speed of 80 km/h.\n\nFind the average speed of the car during this trip. Give your answer in km/h.",
        "answer": "72",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10056,
        "difficulty": null
      },
      {
        "id": "a5d75249c388c9a0fe32cae81e9227a74458b7dbdadab169a2d1d6c88dbf8938",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty}\\left(\\frac{1+5+9+13+\\ldots+(4 n-3)}{n+1}-\\frac{4 n+1}{2}\\right)\n$$",
        "answer": "-\\frac{7}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 46576,
        "difficulty": null
      },
      {
        "id": "9115dfb7ee15409583ca7ec82e894df127a9959dbfa7bbba1dda40283d35b04c",
        "domain": "numina",
        "question": "3. (3 points) Definition: $a \\oplus b=a+b+a b$, then $(2 \\oplus 3) \\oplus 4$ is\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "59",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 139506,
        "difficulty": null
      },
      {
        "id": "ae3add8ab4cd5716b03ebc8ef5914a97f39164401d1a07b36e0890744f83e23d",
        "domain": "numina",
        "question": "38. A flour mill has 20 workers. To gain more profit, it adds a noodle processing project, using the flour produced by the mill to process into noodles (producing $1 \\mathrm{~kg}$ of noodles requires $1 \\mathrm{~kg}$ of flour). It is known that each worker can produce an average of $600 \\mathrm{~kg}$ of flour or process $400 \\mathrm{~kg}$ of noodles per day. Selling flour directly yields a profit of 0.2 yuan per kilogram, while processing it into noodles and selling yields a profit of 0.6 yuan per kilogram. If each worker can only do one job per day, and other factors are not considered, when $\\qquad$ workers process noodles, the mill's total profit per day is maximized, and the maximum profit is $\\qquad$ yuan.",
        "answer": "2880",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100544,
        "difficulty": null
      },
      {
        "id": "f2ee4bcdab556ab72debfeb993d3ce91fea3d5e21caa28723c10b4e4b7b097e9",
        "domain": "numina",
        "question": "On an $n \\times k$ chessboard ( $n, k \\geq 2$ ), some figures are placed. (A figure can only stand on a square, and at most one figure can stand on each square.) We know that on the 4 squares where any 2 rows and 2 columns intersect, there is always exactly 1 or 3 figures. What are the limits within which the number of figures can vary?",
        "answer": "13",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 165554,
        "difficulty": null
      },
      {
        "id": "199a7f025b2822e9eee19f591323fd0dfad23cf0176c4ac417f0fb7d0b36f0cc",
        "domain": "numina",
        "question": "3. A black box contains 60 wooden blocks of the same size and shape, with every 15 blocks painted the same color. To ensure that at least 2 blocks have the same color, one must take out at least $\\qquad$ blocks.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 134714,
        "difficulty": null
      },
      {
        "id": "358b1903d86fb49e47a86589594e4b5ac8a7548a96ada31c64a7e41c1ae4b7b0",
        "domain": "numina",
        "question": "Given the rectangle $A B C D$. On the rays $A B, B C, C D$, and $D A$ starting from $A, B, C$, and $D$ respectively, we mark the points $A_{1}, B_{1}, C_{1}, D_{1}$ such that\n\n$$\n\\frac{A A_{1}}{A B}=\\frac{B B_{1}}{B C}=\\frac{C C_{1}}{C D}=\\frac{D D_{1}}{D A}=k>0\n$$\n\nFor what value of $k$ will the area of the quadrilateral $A_{1} B_{1} C_{1} D_{1}$ be minimal?",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166713,
        "difficulty": null
      },
      {
        "id": "ab24033b2842e27727fb7401350c0e106d2ff60299bb853c77de40f830b84be8",
        "domain": "numina",
        "question": "5. The function $f(x)=a^{2 x}+3 a^{x}-2(a>0, a \\neq 1)$ has a maximum value of 8 on the interval $x \\in[-1,1]$, then its minimum value on this interval is $\\qquad$",
        "answer": "-\\frac{1}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120493,
        "difficulty": null
      },
      {
        "id": "1fc56766c0b7edddeb9ad84ad28d4ad8682982539927b296c33e6b6c32070503",
        "domain": "numina",
        "question": "10.4. Real numbers ${ }^{a}$ and ${ }^{b}$ are such that $a^{3}+b^{3}=1-3 a b$. Find all values that the sum ${ }^{a+b}$ can take.",
        "answer": "-21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9658,
        "difficulty": null
      },
      {
        "id": "8a90317cee412512f5735c6892f8bc947038c2aac6dc607c8ed4761bd9ad282a",
        "domain": "numina",
        "question": "2.108. $\\left(\\frac{a}{b}+\\frac{b}{a}+2\\right) \\cdot\\left(\\frac{a+b}{2 a}-\\frac{b}{a+b}\\right):\\left(\\left(a+2 b+\\frac{b^{2}}{a}\\right) \\cdot\\left(\\frac{a}{a+b}+\\frac{b}{a-b}\\right)\\right) ;$ $a=0.75 ; b=4 / 3$.",
        "answer": "-\\frac{7}{24}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50459,
        "difficulty": null
      },
      {
        "id": "cbd6c6eb15fb7c8573cdfa2d91da60ea7cce19ec10b07159bf46d81eeec7b59d",
        "domain": "numina",
        "question": "Determine the value of $123456123456 \\div 10000001$.",
        "answer": "123456",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74329,
        "difficulty": null
      },
      {
        "id": "9a0efae365552e4da1b3afa18955b5ba51e81eefb02aa9e19469db8e9aa57778",
        "domain": "numina",
        "question": "The ninety-eighth problem: Given a positive integer $\\mathrm{m} \\geq 3$. There are $2 \\mathrm{~m}$ players participating in a round-robin tournament. Each round, the $2 \\mathrm{~m}$ players are divided into $\\mathrm{m}$ groups, with two players competing in each group. In the next round, the players are regrouped to compete again, for a total of $2 m-1$ rounds, ensuring that each player competes against each of the other $2 m-1$ players exactly once. Find the smallest positive integer $n$ such that there exists a feasible tournament schedule where, after $n$ rounds, for any four players, they have either not competed against each other at all or have competed against each other at least twice in total.",
        "answer": "-1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146970,
        "difficulty": null
      },
      {
        "id": "4d291bcf65c44e19acfc85bcbea20f0b085460c65084c43506cd7b4aff9b66b5",
        "domain": "numina",
        "question": "9.2. The bisectors of the external angles $B$ and $C$ of triangle $ABC$ intersect at point $M$. a) Can angle $BMC$ be obtuse? b) Find angle $BAC$ if it is known that $\\angle BMC = \\frac{\\angle BAM}{2}$.",
        "answer": "120",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22561,
        "difficulty": null
      },
      {
        "id": "c3aef5eb5c351ebd00917be857f4c8e5d61b855226dda636eb546d82dea2c136",
        "domain": "numina",
        "question": "Aline tore a sheet of paper into three pieces. Then, she tore one of these pieces into three pieces again; this gave her a total of five pieces. She then repeated the process, tearing one of the five pieces into three pieces, and so on. Can she, at some point, end up with a total of 2020 pieces? 2023 pieces?",
        "answer": "2023",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 81547,
        "difficulty": null
      },
      {
        "id": "ebd6684455179804cee38e44e3dfb6ea3129db29a90d93233b0a7e92beead202",
        "domain": "numina",
        "question": "1. (8 points) Calculate $82-38+49-51=$ $\\qquad$",
        "answer": "42",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119577,
        "difficulty": null
      },
      {
        "id": "42ed3e1524e74c97892e334671b197b68e3819f7847a1afa819c280fdfabc7b5",
        "domain": "numina",
        "question": "What is the value of $123456^{2}-123455 \\times 123457$ ?",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194192,
        "difficulty": null
      },
      {
        "id": "c80eaba1701acddc5b60156d29deda1249bf28d8cf199196e33f299f9ac4c287",
        "domain": "numina",
        "question": "15. A student, in order to plot the graph of the function $y=a x^{2}+b x+c(a \\neq 0)$, took 7 values of the independent variable: $x_{1}<$ $x_{2}<\\cdots<x_{7}$, and $x_{2}-x_{1}=x_{3}-x_{2}=\\cdots=x_{7}-x_{6}$, and calculated the corresponding $y$ values. The following table is listed.\n\\begin{tabular}{|c|c|c|c|c|c|c|c|}\n\\hline$x$ & $x_{1}$ & $x_{2}$ & $x_{3}$ & $x_{4}$ & $x_{5}$ & $x_{6}$ & $x_{7}$ \\\\\n\\hline$y$ & 51 & 107 & 185 & 285 & 407 & 549 & 717 \\\\\n\\hline\n\\end{tabular}\n\nBut due to carelessness, one of the $y$ values was calculated incorrectly. Please point out which value is incorrect and what the correct value should be. Explain your reasoning.",
        "answer": "551",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131147,
        "difficulty": null
      },
      {
        "id": "70780d6e7864346038e4fea67011f4798d601bd322582a30f1c2771f6f798d2b",
        "domain": "numina",
        "question": "# Problem 4. (3 points)\n\nPetya wrote the number 11234567 on the board, and then all the numbers obtained from it by rearranging the digits, in ascending order. What was the position of the number $46753211$?",
        "answer": "12240",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20444,
        "difficulty": null
      },
      {
        "id": "85133125052a847f0bbae926b7e8cf511506006832b77090cf6871d209894490",
        "domain": "numina",
        "question": "3. How many diagonals in a regular 32-sided polygon are not parallel to any of its sides",
        "answer": "240",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6111,
        "difficulty": null
      },
      {
        "id": "7a48f7e14c1f9b2fc0f28a712adca4455067c5a3c1d2100ff815c3a988977627",
        "domain": "numina",
        "question": "What is the largest integer $n$ for which\n\n$$\n\\frac{\\sqrt{7}+2 \\sqrt{n}}{2 \\sqrt{7}-\\sqrt{n}}\n$$\n\nis an integer?",
        "answer": "343",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 168683,
        "difficulty": null
      },
      {
        "id": "28e32a6b0795851d465bedfb0b83d958dafcdaf0b9f838b91c62fd0bf09f9468",
        "domain": "numina",
        "question": "18. Let $x$ and $y$ be real numbers satisfying the inequality\n$$\n5 x^{2}+y^{2}-4 x y+24 \\leq 10 x-1 \\text {. }\n$$\n\nFind the value of $x^{2}+y^{2}$.",
        "answer": "125",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160885,
        "difficulty": null
      },
      {
        "id": "7bab7cd85f28c2ace1223dd22dcddb2ab290ea286dc5be2c743dad8d526c8ed3",
        "domain": "numina",
        "question": "10,11\n\nConstruct a circle equidistant from four points on a plane. How many solutions does the problem have?",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65123,
        "difficulty": null
      },
      {
        "id": "d84a0fcf08e5fefa1bb5da51c8f6c4bd1fad4b3396e5ac556c0032679526b9f6",
        "domain": "numina",
        "question": "78. How many circles of radius 1 are needed to cover a square with a side length of $2$?",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 35836,
        "difficulty": null
      },
      {
        "id": "3d7dd250e723750b425e4f33883e825c55714fa00be0b4bbd28aeab0187d5283",
        "domain": "numina",
        "question": "## Task A-1.5.\n\nIn some language, every word is a string of letters $a$ and $b$. Each word has at least one and at most 13 letters, but not all such strings are words. It is known that by concatenating one word to another, we never get a word. Determine the maximum possible number of words in this language.",
        "answer": "16256",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88275,
        "difficulty": null
      },
      {
        "id": "d35158d2b84eab4876e94de7ca79ff6b49a202bbabe64dc4423bf162eff3fe91",
        "domain": "numina",
        "question": "50th Putnam 1989 Problem B1 S is a square side 1. R is the subset of points closer to its center than to any side. Find the area of R. Solution",
        "answer": "\\frac{4\\sqrt{2}-5}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147309,
        "difficulty": null
      },
      {
        "id": "f3eab4e03079a4b75bff2f51c23da7f9047ac9709774648c8dcd8d05960b7f87",
        "domain": "numina",
        "question": "8. (15 points) Density is defined as the ratio of the mass of a body to its volume. There are two cubes. The second cube is made of a material with twice the density of the first, and the side length of the second cube is 100% greater than the side length of the first. By what percentage is the mass of the second cube greater than the mass of the first?",
        "answer": "1500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4225,
        "difficulty": null
      },
      {
        "id": "83f8a15124665256aee996ca52726deebf912a124193c96fa464653a16e59ef4",
        "domain": "numina",
        "question": "37th Swedish 1997 Problem 4 A and B play a game as follows. Each throws a dice. Suppose A gets x and B gets y. If x and y have the same parity, then A wins. If not, then they make a list of all two digit numbers ab ≤ xy with 1 ≤ a, b ≤ 6. Then they take turns (starting with A) replacing two numbers on the list by their (non-negative) difference. When just one number remains, it is compared to x. If it has the same parity A wins, otherwise B wins. Find the probability that A wins.",
        "answer": "\\frac{3}{4}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 148107,
        "difficulty": null
      },
      {
        "id": "0d16cbc5e42b0f02864516d733dd63cbaf87cbc64a5da20b7c6f019df063b202",
        "domain": "numina",
        "question": "5. Given $n!=n \\times(n-1) \\times(n-2) \\times \\cdots \\times 2 \\times 1$, then $10!\\div(5!\\times 2!)=$",
        "answer": "15120",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 111913,
        "difficulty": null
      },
      {
        "id": "369306b0060c1d0625263fbf61d77b2307fec65fb622db89a638a147f8d77caf",
        "domain": "numina",
        "question": "Problem 11.8. Given a parallelepiped $A B C D A_{1} B_{1} C_{1} D_{1}$. A point $X$ is chosen on the edge $A_{1} D_{1}$, and a point $Y$ is chosen on the edge $B C$. It is known that $A_{1} X=5, B Y=3, B_{1} C_{1}=14$. The plane $C_{1} X Y$ intersects the ray $D A$ at point $Z$. Find $D Z$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-46.jpg?height=501&width=678&top_left_y=359&top_left_x=388)",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15974,
        "difficulty": null
      },
      {
        "id": "7d00ea00862818f9e30b00cab453c7ad8c4cae70cbc5dae0b678f98b8920c3bc",
        "domain": "numina",
        "question": "3. Solve the equation $2021 \\cdot \\sqrt[202]{x^{2020}}-1=2020 x$ for $x \\geq 0 \\cdot$ (10 points)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19075,
        "difficulty": null
      },
      {
        "id": "ac55c09adba45faf7f5a9a1cb61109cb3dfabae544352c7b25d2432ecdaf41b9",
        "domain": "numina",
        "question": "Cut a $4 \\times 4 \\times 4$ cube into 64 $1 \\times 1 \\times 1$ small cubes, then dye 16 of the $1 \\times 1 \\times 1$ small cubes red, requiring that among any 4 small cubes parallel to any edge, exactly 1 small cube is dyed red. The number of different dyeing methods is $\\qquad$ (dyeing methods that are the same after rotation are also considered different dyeing methods).",
        "answer": "576",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125343,
        "difficulty": null
      },
      {
        "id": "c8456e7f20d5ac7d277de0a512599292284f61f2180c4bc9328ec1c30c11cd95",
        "domain": "numina",
        "question": "2. (5 points) At the World Meteorologists Conference, each participant in turn announced the average monthly temperature in their hometown. At this moment, all the others recorded the product of the temperatures in their and the speaker's cities. In total, 78 positive and 54 negative numbers were recorded. What is the smallest number of times a positive temperature could have been announced?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19659,
        "difficulty": null
      },
      {
        "id": "a1c9c8851cd894b81e78a0d78830ef6ef601b822866885d2643cf0b03b938aea",
        "domain": "numina",
        "question": "Two rods - one made of steel, the other of zinc - need to be sized such that at $0^{\\circ}$ C, $a=1 \\mathrm{~m}$, and at $100^{\\circ}$ C, their combined length is $b=1.0024 \\mathrm{~m}$. What is the length of each rod at $0^{\\circ}$ C, given that the linear expansion coefficient of steel is $\\lambda^{\\prime}=0.000011$, and that of zinc is $\\lambda^{\\prime \\prime}=0.000031$?",
        "answer": "0.35",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 175704,
        "difficulty": null
      },
      {
        "id": "0ec71bc52ef82ddf6efb58468ec9758f2ab64b96417d1422902f7221397d0ef8",
        "domain": "numina",
        "question": "### 5.24. Investigate the function\n\n$$\nf(x)=\\frac{5^{1 / x}}{1+5^{1 / x}}\n$$\n\nfor continuity and determine the nature of the points of discontinuity.",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33352,
        "difficulty": null
      },
      {
        "id": "e2d29e1130c8cf801aea36a8b057fbc133a75456bbaf2bb3db8c03da0443ea93",
        "domain": "numina",
        "question": "(11) (25 points) The sum of $n$ positive integers $x_{1}, x_{2}, \\cdots, x_{n}$ is 2009. If these $n$ numbers can be divided into 41 groups with equal sums and also into 49 groups with equal sums, find the minimum value of $n$.",
        "answer": "89",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 134688,
        "difficulty": null
      },
      {
        "id": "75c65571a7b945485c64737c7b83cd25d74e144e5496e809066b5bb5ab35e3bd",
        "domain": "numina",
        "question": "For any sequence of real numbers $A=\\left\\{a_{1}, a_{2}, \\ldots\\right\\}$, we define $\\Delta A$ as the sequence $\\left\\{a_{2}-a_{1}, a_{3}-a_{2}, \\ldots\\right\\}$.\n\nWe assume that all terms of the sequence $\\Delta(\\Delta A)$ are 1 and that $a_{19}=a_{92}=0$.\n\nDetermine $a_{1}$.",
        "answer": "819",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 84714,
        "difficulty": null
      },
      {
        "id": "88210e1bd3484857cded808ec24d4f4c8193f8868661f3c10ca686173dbba345",
        "domain": "numina",
        "question": "Count the number of triangles in the figure: $\\qquad$ triangles.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "11",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137169,
        "difficulty": null
      },
      {
        "id": "db7dac5d9dbd21b639833bb729a0ccf56e8a4da2799c6e4ab870946995f616dc",
        "domain": "numina",
        "question": "A $\\square\\{1,2,3, \\ldots, 49\\}$ does not contain six consecutive integers. Find the largest possible value of |A|. How many such subsets are there (of the maximum size)?\n\n## Answer\n\n$\\max =41$; no. ways 495",
        "answer": "495",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 197461,
        "difficulty": null
      },
      {
        "id": "6f43d92e3b39cf3b3c73324e35be509d2b6902f17f4f870715557bfdf26ad257",
        "domain": "numina",
        "question": "4. Determine the smallest natural number $n$ with the following property: If we choose $n$ different natural numbers less than 2006, there are two among them such that the ratio of the sum and the difference of their squares is greater than three.",
        "answer": "21",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75941,
        "difficulty": null
      },
      {
        "id": "22441c4ea909315cc94007eb850b295c9cdf5f8b5c708913595acb43d0038f57",
        "domain": "numina",
        "question": "12. For any positive integer $n, Z(n)$ is the smallest positive integer $m$ such that $1+2+\\cdots+m$ is a multiple of $n$. Then the sum of all positive integers $n$ that satisfy $Z(n)=6$ is $\\qquad$.",
        "answer": "28",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113738,
        "difficulty": null
      },
      {
        "id": "e1e68b5972294d02e787b45ec833a48e051759f000e8051e493890a4254589da",
        "domain": "numina",
        "question": "48. Among the 2019 natural numbers from 1 to 2019, if any 809 odd numbers are taken and multiplied together, what is the unit digit of the product?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 130649,
        "difficulty": null
      },
      {
        "id": "10bff7a2d36095bfe704594dbfcc7de2c9fc9a112c5075d8b56a9b2c8b987a2f",
        "domain": "numina",
        "question": "10. (6 points) As shown in the figure, the area of the small square is 1, then the area of the shaded part in the figure is\n\n\n\n保留了源文本的换行和格式，这里的翻译结果已经直接输出。",
        "answer": "37.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127981,
        "difficulty": null
      },
      {
        "id": "03a7a3a5c6ff294008e4c32a998ee0c3e8212901eba7af0b270efd5050943e61",
        "domain": "numina",
        "question": "Right triangle $A B C$ is divided by the height $C D$, drawn to the hypotenuse, into two triangles: $B C D$ and $A C D$. The radii of the circles inscribed in these triangles are 4 and 3, respectively. Find the radius of the circle inscribed in triangle $A B C$.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67476,
        "difficulty": null
      },
      {
        "id": "840fb838559b0e0f6288e298b6077e25339e8bfb33c3e5ecd1af830c28322bab",
        "domain": "numina",
        "question": "8. Given $a, b \\in Z$, and $a+b$ is a root of the equation $x^{2}+a x+b=0$, then the maximum possible value of $b$ is",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124106,
        "difficulty": null
      },
      {
        "id": "de5c9d34787c85dcda28d09f72fb4fddaa7c9f6d521831f09bd503374acf6fd9",
        "domain": "numina",
        "question": "1st APMO 1989 Problem 3 ABC is a triangle. X lies on the segment AB so that AX/AB = 1/4. CX intersects the median from A at A' and the median from B at B''. Points B', C', A'', C'' are defined similarly. Find the area of the triangle A''B''C'' divided by the area of the triangle A'B'C'. Solution",
        "answer": "\\frac{25}{49}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147533,
        "difficulty": null
      },
      {
        "id": "d00505019cce3c8a814b9bf9d651ef87180c65e7323aee48b8589e1cda60839e",
        "domain": "numina",
        "question": "Problem 1. The probability that a shooter, upon firing a shot, will score 10 points is 0.4; 9 points - 0.3; and finally, 8 or fewer points - 0.3. Find the probability that the shooter will score no less than 9 points in one shot.",
        "answer": "0.7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 31394,
        "difficulty": null
      },
      {
        "id": "575112f98330e8e27241dd35bc03996ce23fee6cdde4cba6030581262d014be3",
        "domain": "numina",
        "question": "2. Given a positive integer $N<10^{2020}$, when 7 is placed at the first position of $N$, the resulting number is 5 times the number formed when 7 is placed at the last position of $N$. Then the number of all different values of $N$ is $\\qquad$.",
        "answer": "336",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105885,
        "difficulty": null
      },
      {
        "id": "b072dec0845fa76b7e459e06a1295638cc02854676e8622c745a802a5c87cdc0",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{h \\rightarrow 0} \\frac{a^{x+h}+a^{x-h}-2 a^{x}}{h}$",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47123,
        "difficulty": null
      },
      {
        "id": "b137578402ae627deac5332587cdacc91dd79b6a7e5e0fb2717af94a08d784c0",
        "domain": "numina",
        "question": "1. (USS) Three problems $A, B$, and $C$ were given on a mathematics olympiad. All 25 students solved at least one of these problems. The number of students who solved $B$ and not $A$ is twice the number of students who solved $C$ and not $A$. The number of students who solved only $A$ is greater by 1 than the number of students who along with $A$ solved at least one other problem. Among the students who solved only one problem, half solved $A$. How many students solved only $B$ ?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 159099,
        "difficulty": null
      },
      {
        "id": "f7148596bc3d36b4df2c4daf0e5d0fbf1e7a1a1d183543331aafa84d527084c0",
        "domain": "numina",
        "question": "$\\begin{array}{r}\\text { I. (40 points) Given } x_{i} \\in \\mathbf{R}(1 \\leqslant i \\leqslant 2020), \\\\ \\text { and } x_{1010}=1 \\text {. Find the minimum value of } \\sum_{i, j=1}^{2020} \\min \\{i, j\\} x_{i} x_{j} \\text { . }\\end{array}$",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 137020,
        "difficulty": null
      },
      {
        "id": "aa5f219ff48e3e5a00b89aca345d0123366e95deec2cfe2097648bd7a46ff50e",
        "domain": "numina",
        "question": "3. A traveler visited a village where every person either always tells the truth or always lies. The villagers stood in a circle, and each one told the traveler whether the neighbor to their right was truthful or a liar. Based on these statements, the traveler was able to determine unequivocally what fraction of the village's population consists of truthful people. Determine what this fraction is.",
        "answer": "\\frac{1}{2}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 25862,
        "difficulty": null
      },
      {
        "id": "42d926c6af2b340154bb46aa873e04ef3175e6a2d05cbe55208d300be17cfdc0",
        "domain": "numina",
        "question": "10-5. For each natural number from 1 to 999, Damir subtracted the last digit from the first digit and wrote all 1000 differences on the board. For example, for the number 7, Damir wrote the number 0 on the board, for the number 105 he wrote $(-4)$, and for the number 61 he wrote 5.\n\nWhat is the sum of all the numbers on the board",
        "answer": "495",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12388,
        "difficulty": null
      },
      {
        "id": "e2abedf84009d906c750b2cb07a29d9ea5f006c5ff67ce057638a5c8589ad4a4",
        "domain": "numina",
        "question": "B-bank issues ATM cards with a four-digit PIN code that does not contain the digit 0. Mr. Sclerosis was afraid he would forget the PIN code of his card, so he wrote it directly on it, but in Roman numerals IIIVIIIXIV, to make it harder for a potential thief. He revealed his idea to his best friend, Mr. Odkoukal, who was also a client of B-bank. Soon, Mr. Odkoukal did the same with his PIN code and wrote IVIIIVI on his card. To his great surprise, however, he could not determine his PIN code exactly from the Roman numeral notation.\n\n1. What is the PIN code on Mr. Sclerosis's card?\n2. What PIN code could Mr. Odkoukal's card have?",
        "answer": "3794",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76432,
        "difficulty": null
      },
      {
        "id": "2956ed679bdbfd993689943f2dcf65cf3fed7c04293cbf09c8837b762e2aba6a",
        "domain": "numina",
        "question": "8. The sum of 2020 consecutive natural numbers arranged in ascending order is $1010 \\times 2043$. Spider-Man adds the 2nd, 4th, 6th, ..., 2018th, and 2020th numbers, and then divides the sum by 505. The quotient is $\\qquad$ .",
        "answer": "2044",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113612,
        "difficulty": null
      },
      {
        "id": "0c2bc4c59f9e02afd60ec5a22d9dfa020ed441ad74712025edd27b392c80a766",
        "domain": "numina",
        "question": "Question 11: Given the mapping f: $\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ is both injective and surjective, and satisfies $f(x)+f(f(x))=6$, then $f(1)=$ $\\qquad$ _.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95391,
        "difficulty": null
      },
      {
        "id": "dd8bd3be6f6ebfd13ab2fc4fc9b6ad40ae65fa060bfc55b9c1960a45878427cc",
        "domain": "numina",
        "question": "1.598 girls and 25 boys stand in a circle, with at least two boys standing between any two girls. How many different arrangements are there (arrangements that can be made to coincide by rotating the circle are considered the same)?",
        "answer": "\\frac{16!25!}{9!}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 136903,
        "difficulty": null
      },
      {
        "id": "b3a69c6056d54671e4fe2553df619167b119229eaf677c85d62a85c8abf59f33",
        "domain": "numina",
        "question": "Determine a value of $n$ for which the number $2^{8}+2^{11}+2^{n}$ is a perfect square.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74383,
        "difficulty": null
      },
      {
        "id": "64de60eeaaa6b4b4f1877e193d6c35547e07c55db966280f2a2f7c0394a96a50",
        "domain": "numina",
        "question": "2. Let $A=\\overline{a b c b a}$ be a five-digit symmetric number, $a \\neq 0$. If $1 \\leq a \\leq 8$, then the last digit of the number $A+11$ will be $a+1$, and therefore the first digit in the representation of $A+11$ should also be $a+1$. This is possible only with a carry-over from the digit, i.e., when $b=c=9$. Then $A+11=(a+1) 999(a+1)$ is a symmetric number for any $a=1,2, \\ldots, 8$. The case $a=9$ is impossible, since $A+11$ ends in zero, and thus, due to symmetry, it should start with zero. But a number cannot start with zero.\n\nThe total number of solutions is equal to the number of possible choices for the number $a$, which is eight.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 2096,
        "difficulty": null
      },
      {
        "id": "94d24755bbd328fb90f26e40182dc47c7564a01b6300405c53e583142c95d29f",
        "domain": "numina",
        "question": "Exercise 2. How many 8-digit numbers are there whose decimal notation is of the form ab2019cd with a $>0$, and which are divisible by 360?",
        "answer": "20",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78145,
        "difficulty": null
      },
      {
        "id": "0a19c8eff9b55c59ae658f7effea66b50c090a6f511a20b317bd99e1c8b4ccce",
        "domain": "numina",
        "question": "4. A frog jumps over lily pads. It can jump 7 lily pads to the right or 4 lily pads to the left. What is the minimum number of jumps needed for the frog to move exactly 2024 lily pads to the right?\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_1d05bbb73aad96dd359ag-3.jpg?height=285&width=1514&top_left_y=1945&top_left_x=294)",
        "answer": "297",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86498,
        "difficulty": null
      },
      {
        "id": "10929ded64e2d8d4deec257458ab3efd11ff951d6d5499d92878fc07ae7c6738",
        "domain": "numina",
        "question": "3. Let $n$ be an integer greater than 1 . If $n$ is greater than 1200 times any one of its prime factors, find the smallest possible value of $n$.\n(1 mark)\n設 $n$ 為大於 1 的整數。若 $n$ 較它的任何一個質因數的 1200 倍為大, 求 $n$ 的最小可能值。",
        "answer": "3888",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153613,
        "difficulty": null
      },
      {
        "id": "dd93c8647d054152a4096b337c4295c9ea1e8481b8ad773d9f2e6dc03b263fbf",
        "domain": "numina",
        "question": "I3.2 Two bags $U_{1}$ and $U_{2}$ contain identical red and white balls. $U_{1}$ contains $A$ red balls and 2 white balls. $U_{2}$ contains 2 red balls and $B$ white balls. Take two balls out of each bag. If the probability of all four balls are red is $\\frac{1}{60}$, find the value of $B$.",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153108,
        "difficulty": null
      },
      {
        "id": "36ec4cda858d1116e5f388458321d8266ae3db6207006aecaad6f667094c725f",
        "domain": "numina",
        "question": "14. Two circles $C_{1}$ and $C_{2}$ with centers $A$ and $B$ are externally tangent at $T$. Let $B D$ be a segment tangent to $C_{1}$ at $D$ and let $T C$ be the segment tangent to both at $T$ with $C \\in B D$. If $A T$ is 80 units long and $B T$ is 90 units long, what is the length of $C D$?",
        "answer": "48",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 181355,
        "difficulty": null
      },
      {
        "id": "76f5c4db02d919b683fc3569cfffc927e61d79cafe3ae6565971a7b4eb3ee3ac",
        "domain": "numina",
        "question": "5. July 1, 2021 is the 100th anniversary of the founding of our Party, recorded as 20210701, where the sum of the digits in the odd positions is $1+7+1+0=9$ and the sum of the digits in the even positions is $0+0+2+2=4$, with a ratio of $9: 4$; among the 8-digit numbers formed by $0 \\sim 9$ (no repetition), how many satisfy the condition that the sum of the digits in the odd positions to the sum of the digits in the even positions is $9: 4$?",
        "answer": "2304",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132400,
        "difficulty": null
      },
      {
        "id": "24b18ecf48a863c805127e13ba08d14f1b13fd0db97a0e1f9d6c54c754d160e0",
        "domain": "numina",
        "question": "358. Calculate the integral $\\int_{1}^{9} \\sqrt{6 x-5} d x$ using the Newton-Leibniz formula and approximate formulas for rectangles and trapezoids, dividing the interval of integration into 8 equal parts. Estimate the error of the results.",
        "answer": "38",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32196,
        "difficulty": null
      },
      {
        "id": "787bcd257d52adc933c979af3bf519459dbe065cf19c136242e5d96e37f393d4",
        "domain": "numina",
        "question": "1. Fold the triangular paper piece $A B C$ along a line parallel to $B C$, the folded part intersects $B C$ at points $F, G$. Given $B C=9 \\mathrm{~cm}, F G=3 \\mathrm{~cm}$, then the length of the crease $D E$ is $\\qquad$ $\\mathrm{cm}$.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 121535,
        "difficulty": null
      },
      {
        "id": "9be7fdbe86bd086ffd8c927fd005c17fd18dd76bcac001ef5baaaf6b8cffd687",
        "domain": "numina",
        "question": "10. The function $y=f(x)$ defined on $\\mathbf{R}$ has the following properties:\n（1）For any $x \\in \\mathbf{R}$, $f\\left(x^{3}\\right)=f^{3}(x)$;\n（2）For any $x_{1} 、 x_{2} \\in \\mathbf{R}, x_{1} \\neq x_{2}$, $f\\left(x_{1}\\right) \\neq f\\left(x_{2}\\right)$. Then the value of $f(0)+f(1)+f(-1)$ is $\\qquad$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144471,
        "difficulty": null
      },
      {
        "id": "7e297effb48497b3aabceaa626c640ff863df2f1e52fa85b1ff3a11623dab407",
        "domain": "numina",
        "question": "20. 11, 12, 13 are three consecutive natural numbers, the sum of their digits is $1+1+1+2+1+3=9$. There are three consecutive natural numbers, all less than 100, the sum of their digits is 18, there are $\\qquad$ possible cases.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 131068,
        "difficulty": null
      },
      {
        "id": "6fc082f53d1a5b36993d73b53826f0d0ecacb2409079ad07ecca001fb6b0f951",
        "domain": "numina",
        "question": "5. (10 points) The 3rd term of an arithmetic sequence is 14, and the 18th term is 23. How many terms in the first 2010 terms of this sequence are integers.",
        "answer": "402",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96602,
        "difficulty": null
      },
      {
        "id": "c7b1195148e21d66f22856c2e3e58d7f1282aa9fb0ef9edb80ef063393eca523",
        "domain": "numina",
        "question": "9. Given that two cars, A and B, start from points $A$ and $B$ respectively at the same time, and travel back and forth between $A$ and $B$ at a constant speed. If after the first meeting, car A continues to drive for 4 hours to reach $B$, while car B only drives for 1 hour to reach $A$, then when the two cars meet for the 15th time (excluding meetings at points $A$ and $B$), they have driven $\\qquad$ hours.",
        "answer": "86",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93490,
        "difficulty": null
      },
      {
        "id": "85002677125c2a9e2752f5af785fa8faa6b0eb41ffc13637b9773932216eec87",
        "domain": "numina",
        "question": "13th Iberoamerican 1998 Problem A3 Find the smallest number n such that given any n distinct numbers from {1, 2, 3, ... , 999}, one can choose four different numbers a, b, c, d such that a + 2b + 3c = d. Solution",
        "answer": "835",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 147897,
        "difficulty": null
      },
      {
        "id": "2fe9fe5b2a4f340fc599eac0999169f0f1985028fcb7bdea795b70fead9dbad8",
        "domain": "numina",
        "question": "Example 9 In $\\triangle A B C$, it is known that point $E$ is the midpoint of side $B C$, $D$ is on $A C$, $A C=1, \\angle B A C=$ $60^{\\circ}, \\angle A B C=10^{\\circ}, \\angle D E C=80^{\\circ}$, find the value of $S_{\\triangle A B C}+2 S_{\\triangle C D E}$.",
        "answer": "\\frac{\\sqrt{3}}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 145575,
        "difficulty": null
      },
      {
        "id": "6a91235e26c58e3075051d53e031fa780c76bd6c9d09339dcb748a56f9dcb3c1",
        "domain": "numina",
        "question": "37. The area of the triangle formed by the lines $y=x+2$, $y=-2x+5$ and the $x$-axis is equal to",
        "answer": "\\frac{27}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 104371,
        "difficulty": null
      },
      {
        "id": "f0fe0ab51c29f9af8953c0a1bb558b110e2dde2d5adbf28db09abe200dfb99f2",
        "domain": "numina",
        "question": "7. Let $a$ be a non-zero real number, in the Cartesian coordinate system $x O y$, the focal distance of the quadratic curve $x^{2}+a y^{2}+a^{2}=0$ is 4, then the value of $a$ is $\\qquad$",
        "answer": "\\frac{1-\\sqrt{17}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 123739,
        "difficulty": null
      },
      {
        "id": "02605c94ff9d9090dcf681cdd2b12c47eb1cd4928988d6772deb5d3801cb29fc",
        "domain": "numina",
        "question": "Solve for real $x$ :\n\n$$\nx+\\sqrt{(x+1)(x+2)}+\\sqrt{(x+2)(x+3)}+\\sqrt{(x+3)(x+1)}=4 .\n$$",
        "answer": "-\\frac{311}{840}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90279,
        "difficulty": null
      },
      {
        "id": "0483c3fecae149cc4eae5dacd6b0fd5df6e58fcb68bbcbb40554c8c374524ab7",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{5 / 2} \\frac{x^{2} \\cdot d x}{\\sqrt{25-x^{2}}}\n$$",
        "answer": "\\frac{25\\pi}{12}-\\frac{25\\sqrt{3}}{8}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46816,
        "difficulty": null
      },
      {
        "id": "c84910c4543db39ded308825ef6c606b7eb7eb8510d09e93ebdecdcf56086bfc",
        "domain": "numina",
        "question": "In the diagram, $P A D B, T B C$ and $N C D$ are straight line segments. If $\\angle T B D=110^{\\circ}$, $\\angle B C N=126^{\\circ}$, and $D C=D A$, determine the measure of $\\angle P A C$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_3011c729c030a190170fg-1.jpg?height=398&width=677&top_left_y=1189&top_left_x=781)",
        "answer": "152",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194138,
        "difficulty": null
      },
      {
        "id": "cbf44897fcaf71fc38a3d8cf399be0013779848399e08e99e11f4bc4250d78a1",
        "domain": "numina",
        "question": "8,9\n\nOn a circle, a point $A$ is taken, on the diameter $B C$ - points $D$ and $E$, and on its extension beyond point $B$ - point $F$. Find $B C$, if $\\angle B A D=\\angle A C D, \\angle B A F=\\angle C A E, B D=2, B E=5$ and $B F=4$.",
        "answer": "11",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64893,
        "difficulty": null
      },
      {
        "id": "61fe8515a00c9c92bf19157ee03542b4822e911267809d51ce22bff471caff9b",
        "domain": "numina",
        "question": "3. Let $A B C D$ be a regular tetrahedron with side length 2 . The plane parallel to edges $A B$ and $C D$ and lying halfway between them cuts $A B C D$ into two pieces. Find the surface area of one of these pieces.",
        "answer": "1+2\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152308,
        "difficulty": null
      },
      {
        "id": "06659c3f82e051314e408a780efcbf08daea52f3c361ec6774778445799fee56",
        "domain": "numina",
        "question": "5. Given a convex hexagon $A B C D E F$ with six equal side lengths, the interior angles $\\angle A$, $\\angle B$, and $\\angle C$ are $134^{\\circ}$, $106^{\\circ}$, and $134^{\\circ}$, respectively. Then the measure of the interior angle $\\angle E$ is $\\qquad$.",
        "answer": "134",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105717,
        "difficulty": null
      },
      {
        "id": "6a03b79d13147f7ec923413c917b8374a124532e4a49183bd22ffcd8e7096f78",
        "domain": "numina",
        "question": "In the pyramid $A B C D$, the dihedral angle at edge $A C$ is $90^{\\circ}, A B=B C=C D, B D=A C$. Find the dihedral angle at edge $A D$.\n\n#",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62287,
        "difficulty": null
      },
      {
        "id": "840bdfb0864e4071785b44bad766ffd1ce10ed9d34100b6ebc2953f0a2d47425",
        "domain": "numina",
        "question": "9. Given that a line passing through the focus $F$ of the parabola $y^{2}=4 x$ intersects the parabola at points $M$ and $N$, and $E(m, 0)$ is a point on the $x$-axis. The extensions of $M E$ and $N E$ intersect the parabola at points $P$ and $Q$ respectively. If the slopes $k_{1}$ and $k_{2}$ of $M N$ and $P Q$ satisfy $k_{1}=3 k_{2}$, then the value of the real number $m$ is",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 112563,
        "difficulty": null
      },
      {
        "id": "f043913e81fd718f4399110c0ea95887afcb8714c39e0038f38d10891782b817",
        "domain": "numina",
        "question": "4. Given nine-digit numbers $m$ and $n$, obtained from each other by writing the digits in reverse order. It turned out that the product $mn$ consists of an odd number of digits and reads the same from left to right and from right to left. Find the largest number $m$ for which this is possible.",
        "answer": "220000001",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 8228,
        "difficulty": null
      },
      {
        "id": "3e73a33df4da0000c7bf2d6a0dc510b641c419cf6cec9dd55526c92ca425cbd2",
        "domain": "numina",
        "question": "## Task B-1.2.\n\nIf $a-b=3$, and $a \\cdot b=1$, what is $\\frac{1}{a^{3}}-\\frac{1}{b^{3}}$ and $\\frac{1}{a^{6}}+\\frac{1}{b^{6}}$?",
        "answer": "-361298",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89534,
        "difficulty": null
      },
      {
        "id": "66fda566abc6e758d487bf56fa43b2a6ced35716adf30a2872541875773a8c51",
        "domain": "numina",
        "question": "There are given 100 distinct positive integers. We call a pair of integers among them good if the ratio of its elements is either 2 or 3. What is the maximum number $g$ of good pairs that these 100 numbers can form? (A same number can be used in several pairs.)",
        "answer": "180",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 197345,
        "difficulty": null
      },
      {
        "id": "f8ece06966711cde0f5eb2e455e77ea6057f8e298327594c4c3f721a5a53a4db",
        "domain": "numina",
        "question": "4. A certain three-digit number is a multiple of 2, adding 1 makes it a multiple of 3, adding 2 makes it a multiple of 4, adding 3 makes it a multiple of 5, and adding 4 makes it a multiple of 6. The smallest such number is $\\qquad$ .",
        "answer": "122",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105348,
        "difficulty": null
      },
      {
        "id": "5494933c04fe020e20349e6ccac4547f9f6ebc06b117a00edd3f6cdefaa613d8",
        "domain": "numina",
        "question": "Example 5 Given that $f(x)$ is a function defined on the set of real numbers, and $f(x+2)[1-f(x)]=1+f(x)$, if $f(5)$ $=2+\\sqrt{3}$, find $f(2009)$.",
        "answer": "\\sqrt{3}-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 106441,
        "difficulty": null
      },
      {
        "id": "782c64fcb568561159c8a67a1b2c820681a712ac588c88d1f8b7a08b1a294454",
        "domain": "numina",
        "question": "## Task 27/77\n\nDetermine the largest natural number $z$ that consists of 10 different digits (in the decimal positional system) and is divisible by 11.",
        "answer": "9876524130",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182514,
        "difficulty": null
      },
      {
        "id": "f8b40823e3397eaa2ab1ec1636dc576a4db7393bbb4cab2aff7e5d534747245b",
        "domain": "numina",
        "question": "Problem 12.3. Find the maximal cardinality of a set of phone numbers satisfying the following three conditions:\n\na) all of them are five-digit numbers (the first digit can be 0 );\n\nb) each phone number contains at most two different digits;\n\nc) the deletion of an arbitrary digit in two arbitrary phone numbers (possibly in different positions) does not lead to identical sequences of digits of length 4 .\n\nIvan Landjev",
        "answer": "190",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 177680,
        "difficulty": null
      },
      {
        "id": "d8988fcde04bafad79057c8bdb5ce0a744b56de49dd971bbb19717b7010d96fc",
        "domain": "numina",
        "question": "Example 8. An urn contains 8 red and 6 blue balls. Three balls are drawn sequentially without replacement from the urn. Find the probability that all 3 balls are blue.",
        "answer": "\\frac{5}{91}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32951,
        "difficulty": null
      },
      {
        "id": "86fb3ed2de4c48ff12b368f0b55c793716f1ddde5f668c646527b291b067b8cd",
        "domain": "numina",
        "question": "5. Let the line $y=k x-2$ intersect the parabola $y^{2}=8 x$ at points $A$ and $B$. If the x-coordinate of the midpoint of segment $A B$ is 2, then the length of segment $A B$ is $|A B|=$ $\\qquad$",
        "answer": "2\\sqrt{15}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133902,
        "difficulty": null
      },
      {
        "id": "c41209fba9ac387a56e452e64228f27ddcff48611ad8172b1ada8793c43fcecb",
        "domain": "numina",
        "question": "Task 1. (5 points) Find $\\frac{a^{12}-4096}{64 a^{6}}$, if $\\frac{a^{2}}{4}-\\frac{4}{a^{2}}=5$.",
        "answer": "140",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7211,
        "difficulty": null
      },
      {
        "id": "e4d3b66e07635e9e8e88daf2f1f87492aa7b0d86e8d54eb2480b187bef3a0b66",
        "domain": "numina",
        "question": "7. In the company, there are elves, fairies, and gnomes. Each elf is friends with all fairies except for three, and each fairy is friends with twice as many elves. Each elf is friends with exactly three gnomes, and each fairy is friends with all gnomes. Each gnome is friends with exactly half of the elves and fairies combined. How many gnomes are there in the company?",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21553,
        "difficulty": null
      },
      {
        "id": "aaef5617c80bf774d7acf4cf41a1d04ce3e10d9472452206e04d53a6047dfcd6",
        "domain": "numina",
        "question": "Three. (50 points) Find the smallest positive integer $n$, such that there exist rational numbers $a_{i}, b_{i} (i=1,2, \\cdots, n)$ satisfying\n$$\nx^{2}+x+4=\\sum_{i=1}^{n}\\left(a_{i} x+b_{i}\\right)^{2} .\n$$",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135660,
        "difficulty": null
      },
      {
        "id": "f505c203fd8fd74ee2279ce63fba0008f662aad39862b1bf5a9466560ffa1e67",
        "domain": "numina",
        "question": "17. What is the remainder when\n$$\n16^{15}-8^{15}-4^{15}-2^{15}-1^{15}\n$$\nis divided by 96 ?",
        "answer": "31",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154266,
        "difficulty": null
      },
      {
        "id": "91d508cc60b74a7000fd7b15f9cd2ceb7b54856a714897332f89a1569c4727b8",
        "domain": "numina",
        "question": "5) A class of students is planting a batch of saplings. If each student is given k saplings, there will be 38 left; if each student is allocated 9 saplings, there will be a shortage of 3. Therefore, the class has $\\qquad$ students.",
        "answer": "41",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93389,
        "difficulty": null
      },
      {
        "id": "e47e37f22a9eb095b43ab8ac65f3bb3e260fad5af8b3e532589c8aba991b9f33",
        "domain": "numina",
        "question": "Problem 9.5. Point $M$ is the midpoint of side $B C$ of triangle $A B C$, where $A B=17$, $A C=30, B C=19$. A circle is constructed with side $A B$ as its diameter. An arbitrary point $X$ is chosen on this circle. What is the minimum value that the length of segment $M X$ can take?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_85a336fa11c4e8eb26a1g-03.jpg?height=414&width=553&top_left_y=1517&top_left_x=448)",
        "answer": "6.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17038,
        "difficulty": null
      },
      {
        "id": "3a62cc45c370fa53806497fcfd648b06fdbbd1ea4232ddcad2d5466fc9e28a26",
        "domain": "numina",
        "question": "8. Given the function $f(x)=x^{2}+a x+b, a, b \\in \\mathbf{R}$, let $M(a, b)$ be the maximum value of $|f(x)|$ on the interval $[-1,1]$. When $a, b$ satisfy $M(a, b) \\leqslant 2$, the maximum value of $|a|+|b|$ is $\\qquad$ .",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120088,
        "difficulty": null
      },
      {
        "id": "5d8bde9f646b1718dba58c7c87c5f21d2fcacb56c5c5f189191a956e15e189d7",
        "domain": "numina",
        "question": "Problem 10.6. In a convex quadrilateral $A B C D$, the midpoint of side $A D$ is marked as point $M$. Segments $B M$ and $A C$ intersect at point $O$. It is known that $\\angle A B M=55^{\\circ}, \\angle A M B=$ $70^{\\circ}, \\angle B O C=80^{\\circ}, \\angle A D C=60^{\\circ}$. How many degrees does the angle $B C A$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_564c13f715a760703913g-39.jpg?height=371&width=709&top_left_y=89&top_left_x=372)",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 16923,
        "difficulty": null
      },
      {
        "id": "b6486267c68b6b18b9f0e6eab7c5943b750014b5f0c4cdbb5db0bb3911b57839",
        "domain": "numina",
        "question": "1. (3 points) Calculate: $19 \\times 0.125+281 \\times \\frac{1}{8}+12.5=$",
        "answer": "50",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141398,
        "difficulty": null
      },
      {
        "id": "3db5967fc3fc732b3162dad0db29d53b19688726db8d2ad6c0021f0c1f47a81a",
        "domain": "numina",
        "question": "3. Find the number of positive integer solutions to the equation $x_{1}+x_{2}+x_{3}+x_{4}=23$, with the conditions $x_{1} \\leqslant 9, x_{2} \\leqslant 8, x_{3} \\leqslant 7$, $x_{4} \\leqslant 6$.",
        "answer": "115",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 124154,
        "difficulty": null
      },
      {
        "id": "fc3b4c0cb1a8ff9d7a719be0b8157e3ebb034c480b4e2d1c5f68e35ecd7b83b1",
        "domain": "numina",
        "question": "4. For what values of the parameter $a$ does the system of equations\n\n$$\n\\left\\{\\begin{array}{l}\nx^{2}+y^{2}+z^{2}+2 y=0 \\\\\nx+a y+a z-a=0\n\\end{array}\\right.\n$$\n\nhave a unique solution?",
        "answer": "\\\\frac{\\sqrt{2}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4396,
        "difficulty": null
      },
      {
        "id": "1c39cfa9719475bb8a63342ea471395f31fdc250c17ae70c0ab6eb7f67ee6663",
        "domain": "numina",
        "question": "Example 5. Calculate the double integral $I=\\int_{D} \\int(x+y) d x d y$, where $D-$ is the region bounded by the lines $x=0, y=x^{2}+x-3$, $2 y=3 x(x \\geqslant 0)$.",
        "answer": "\\frac{14}{5}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31267,
        "difficulty": null
      },
      {
        "id": "d49161554aa1b79a9cb15c6de610713e2cbf0a293a81a8f4e0b26ec20f81a470",
        "domain": "numina",
        "question": "9. (10 points) In a certain exam, 153 people scored no more than 30 points, with an average of 24 points; 59 people scored no less than 80 points, with an average of 92 points; those who scored more than 30 points had an average of 62 points, and those who scored less than 80 points had an average of 54 points. How many people in total participated in this exam?",
        "answer": "1007",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112369,
        "difficulty": null
      },
      {
        "id": "b3ede0241747a35f10271aa646f3fd2ddb5abae16623d30b426dde7cbbf227cc",
        "domain": "numina",
        "question": "1. a) Solve in $Z$ the equation: $5 \\cdot(2 \\cdot|3 x-4|+4)-30=10$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178801,
        "difficulty": null
      },
      {
        "id": "dcdfa6ce823e166e7a6529deb7290a6bd9dec7b3039ea8c14d2eb5e9ed9f23a4",
        "domain": "numina",
        "question": "411. Find the curvature of the curve: 1) $x=t^{2}, y=2 t^{3}$ at the point where $t=1$; 2) $y=\\cos 2 x$ at the point where $x=\\frac{\\pi}{2}$.",
        "answer": "4",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 34246,
        "difficulty": null
      },
      {
        "id": "a623f62304f80c00fdde5593633123377d0b84261dd2a9db6c3e201b7d8e5dd0",
        "domain": "numina",
        "question": "Problem 10.8. Given a triangle $A B C$. Let point $I$ be the center of its inscribed circle, and points $P$ and $Q$ be the midpoints of sides $A B$ and $A C$ respectively. It turns out that $\\angle P I Q + \\angle B I C = 180^{\\circ}$. Find the length of segment $B C$, if $A B = 20$ and $A C = 14$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_baaf26a33e2d9a36d9b4g-06.jpg?height=501&width=422&top_left_y=575&top_left_x=516)",
        "answer": "\\frac{34}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13115,
        "difficulty": null
      },
      {
        "id": "fb92e6bb6f45d87d9c1012e09eca811da743123782fe8811f9e5c5861b886005",
        "domain": "numina",
        "question": "8. (10 points) On a plane, use toothpicks of length 6 cm to form squares. To form a square with a side length of 6 cm, 4 toothpicks are needed. To form 5 such squares, at least $\\qquad$ toothpicks are needed.",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127341,
        "difficulty": null
      },
      {
        "id": "7eeba0e26805cb79358144e61d459c7127c59cac1cb1c8f24a9f9ac6b289f41b",
        "domain": "numina",
        "question": "Some Christmas lights are arranged using magical wires. Each light can be green or yellow. Each wire is connected to two lights and has a magical property: when someone touches a wire connecting two lights, each of them changes color from green to yellow or from yellow to green.\n\na) In the arrangement below, each point represents a light and the segments represent the magical wires. At the beginning, all of them are yellow. What is the minimum number of wires we need to touch so that all the lights turn green? Show an example of how to make this change with the minimum number of wires.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-24.jpg?height=320&width=463&top_left_y=859&top_left_x=702)\n\nb) Consider the arrangement in the following figure where all the lights are yellow. Show that it is not possible to touch some magical wires and change the color of all the lights to green.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-24.jpg?height=323&width=685&top_left_y=1426&top_left_x=594)",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 73956,
        "difficulty": null
      },
      {
        "id": "471ff1ddea4245aa4a8e1fa2c9122cb4264c15dd35b3226e69ea4d2ef0b3506b",
        "domain": "numina",
        "question": "$3 \\cdot 33$ Find the sum $\\sum_{n=1}^{1994}(-1)^{n} \\frac{n^{2}+n+1}{n!}$.",
        "answer": "-1+\\frac{1995}{1994!}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95071,
        "difficulty": null
      },
      {
        "id": "80ec2c6246b91fb72f7fcfbed5c939c8ca14bf2ef2df7c3c110a667bc5ea7c60",
        "domain": "numina",
        "question": "We drew all the diagonals in three polygons with different numbers of sides. The second polygon has 3 more sides and 3 times as many diagonals as the first one. The third one has 7 times as many diagonals as the second one. How many more sides does the third polygon have compared to the second one?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 165786,
        "difficulty": null
      },
      {
        "id": "6dc7a4461ce499931fbff19f2bb2c3b368dfa7c057edc9b41d02941a0015ead0",
        "domain": "numina",
        "question": "3. A pedestrian is walking along a path next to a steep track (in the longitudinal direction) at a speed of $6 \\mathrm{~km} / \\mathrm{h}$. He noticed that the train coming from the opposite direction passed him in 12.6 seconds. With the same train, while walking uphill at a speed of $3.6 \\mathrm{~km} / \\mathrm{h}$, he passed it in 15 seconds. What is the length of the train and what is its speed?",
        "answer": "210",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190481,
        "difficulty": null
      },
      {
        "id": "88d8a442ca943654a22fdb0dcfc3a11ac03da029fee244764b72d010954f2498",
        "domain": "numina",
        "question": "7. Let positive real numbers $x, y$ satisfy $x^{2}+y^{2}+\\frac{1}{x}+\\frac{1}{y}=\\frac{27}{4}$. Then the minimum value of $P=\\frac{15}{x}-\\frac{3}{4 y}$ is",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109327,
        "difficulty": null
      },
      {
        "id": "ac0e81dca52db534fde0a58a230b426b8e786138b976e06cc8ee67a3bf28d99b",
        "domain": "numina",
        "question": "With 5 equal digits, I can write 100 as follows: $100=111-11$, or $100=3 \\times 33+\\frac{3}{3}$, or $(5+5+5+5) \\times 5$. Based on these, let's write 100 with 9 different digits.",
        "answer": "38",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 176729,
        "difficulty": null
      },
      {
        "id": "d0251c95dcf7eeba4a721d3dd8f65be71f846e6c9450bb41d30917e2f33281fa",
        "domain": "numina",
        "question": "2. There is a bag with 16 letters: А, А, А, А, В, В, Д, И, И, М, М, Н, Н, Н, Я, Я. Anna, Vanya, Danya, and Dima each took 4 letters from it, after which the bag was empty. How many of them could have successfully spelled out their names? Explain your answer.",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13935,
        "difficulty": null
      },
      {
        "id": "5424f755345cb652d2134b872c80bf8423bb36e539592d1a3fce15a16417c61c",
        "domain": "numina",
        "question": "10. In Rt $\\triangle A B C$, $\\angle C=90^{\\circ}, D, E$ are points on sides $B C, A C$ respectively. Given $A D=17$, $B E=13, D E=\\sqrt{97}$, then $A B=$ $\\qquad$ .",
        "answer": "19",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92462,
        "difficulty": null
      },
      {
        "id": "93184aa9ab4cbc2b82717f5be01c5333b786cb8e762a8c83e0c57c2420e9ccfe",
        "domain": "numina",
        "question": "\\section*{Problem 6 - 061046}\n\nGive the total number of all different integer solution pairs \\((x, y)\\) of the inequality\n\n\\[\n|x|+|y| \\leq 100\n\\]\n\nwhere two solution pairs \\(\\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right)\\) are considered the same if and only if \\(x_{1}=x_{2}\\) and \\(y_{1}=y_{2}\\).",
        "answer": "20201",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 184964,
        "difficulty": null
      },
      {
        "id": "fe45948542c89ab44cec88b6b50ecee8d0f54635bb938bd6d84bcdb13ab310da",
        "domain": "numina",
        "question": "G87 (47-2, 塞黑) Let $P$ be a regular 2006-gon. If an end of a diagonal of $P$ divides the boundary of $P$ into two parts, each containing an odd number of sides of $P$, then the diagonal is called a \"good edge\". It is stipulated that each side of $P$ is a \"good edge\".\n\nGiven 2003 non-intersecting diagonals inside $P$ that divide $P$ into several triangles, how many isosceles triangles with two \"good edges\" can there be in such a division?\n\n\n\nTranslating the text into English while preserving the original formatting and line breaks:\n\nG87 (47-2, 塞黑) Let $P$ be a regular 2006-gon. If an end of a diagonal of $P$ divides the boundary of $P$ into two parts, each containing an odd number of sides of $P$, then the diagonal is called a \"good edge\". It is stipulated that each side of $P$ is a \"good edge\".\n\nGiven 2003 non-intersecting diagonals inside $P$ that divide $P$ into several triangles, how many isosceles triangles with two \"good edges\" can there be in such a division?",
        "answer": "1003",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143215,
        "difficulty": null
      },
      {
        "id": "3ea7e806d21463c217176d19d9fa186014d39155f7435920bf0b6dbead1d01d3",
        "domain": "numina",
        "question": "11. (20 points) Given the function\n$$\nf(x)=\\left(1-x^{2}\\right)\\left(x^{2}+b x+c\\right)(x \\in[-1,1]) \\text {. }\n$$\n\nLet the maximum value of $\\mid f(x)$ be $M(b, c)$. When $b$ and $c$ vary, find the minimum value of $M(b, c)$.",
        "answer": "3-2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105054,
        "difficulty": null
      },
      {
        "id": "74341e2775d83ead61b4eb48e358ecb2ceb27324987c94d940596f009e550271",
        "domain": "numina",
        "question": "Task B-1.6. How many ordered pairs of integers $x, y, 0<x<y<10^{6}$ are there such that their arithmetic mean is exactly 8 greater than their geometric mean? (The geometric mean of numbers $a$ and $b$ is the number $\\sqrt{a \\cdot b}$.)",
        "answer": "995",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89304,
        "difficulty": null
      },
      {
        "id": "0270dc50fd5a42b6ae4390c8f9d346c5e1cd0d724e1a29ff058eee95cbbc4a8b",
        "domain": "numina",
        "question": "Ali Baba, upon returning to the cave filled with diamonds and gold, pondered the following: The treasures can only be placed in his sack. From experience, he knows that 40 pounds of diamonds and 200 pounds of gold can fit into it, but he can only carry and take away 100 pounds. In the market, 1 pound of gold can be exchanged for 20 tevets, and 1 pound of diamonds for 60 tevets. How should he fill his sack so that he can buy the maximum number of tevets?",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167269,
        "difficulty": null
      },
      {
        "id": "694b4f39e4882974cfa18cbeb873bbfbdd7cc6703addf975a1e4b4c5b6c05ce2",
        "domain": "numina",
        "question": "Example 3 In the Cartesian coordinate system $x O y$, given two points $M(-1,2)$ and $N(1,4)$, point $P$ moves on the $x$-axis. When $\\angle M P N$ takes the maximum value, find the x-coordinate of point $P$.\n\n untranslated text remains unchanged.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 131944,
        "difficulty": null
      },
      {
        "id": "7a7aea8f68ef2b7b7f9e77d5bff4523f02a47c207db2897348cba3d1f906a658",
        "domain": "numina",
        "question": "2. (8 points) Xiaoming's family keeps three hens, the first hen lays one egg per day, the second hen lays one egg every two days, and the third hen lays one egg every three days. It is known that on January 1st, all three hens laid eggs, so in the thirty-one days of January, these three hens laid $\\qquad$ eggs in total,",
        "answer": "58",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 145001,
        "difficulty": null
      },
      {
        "id": "8b451802e56188a0ec0f2e1741f95f4c725ff369bd25ed4b174215689107ed3b",
        "domain": "numina",
        "question": "2. Each of the 10 people is either a knight, who always tells the truth, or a liar, who always lies. Each of them thought of some natural number. Then the first said: “My number is greater than 1”, the second said: “My number is greater than 2”, ..., the tenth said: “My number is greater than 10”. After that, they, in a different order, said (each one phrase): “My number is less than 1”, “My number is less than 2”, ..., “My number is less than 10”. What is the maximum number of knights that could have been among these 10 people? (O. Podlipsky)",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 6986,
        "difficulty": null
      },
      {
        "id": "e3a4aebcdc850b32604ebbfe6fbe1cef802b17fc9c34d37bbd1c5ce9afa7ded3",
        "domain": "numina",
        "question": "3. In a regular hexagon $A B C D E F$ with side length $a$, the extensions of sides $A F$ and $D E$ intersect at point $T$. Calculate the exact length of segment $B T$.",
        "answer": "\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 163645,
        "difficulty": null
      },
      {
        "id": "5d20d0d9edc2741e9011d38ebb21ce54bea043c610529a67ea74917c52f47b6a",
        "domain": "numina",
        "question": "Variant 11.3.4. The polynomial $G(x)$ with real coefficients takes the value 2022 at exactly five different points $x_{1}<x_{2}<x_{3}<x_{4}<x_{5}$. It is known that the graph of the function $y=G(x)$ is symmetric with respect to the line $x=-6$.\n\n(a) (2 points) Find $x_{1}+x_{3}+x_{5}$.\n\n(b) (2 points) What is the smallest degree that $G(x)$ can have?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12192,
        "difficulty": null
      },
      {
        "id": "1dcfd82b22285e4916667efb5c68418061f4f6d9982ea1b89170ddc039ffa16b",
        "domain": "numina",
        "question": "Two identical equilateral triangles $A B C$ and $C D E$ with side length 1 are positioned such that they share only one common point $C$ and the angle $B C D$ is less than $60^{\\circ}$. Point $K$ is the midpoint of $A C$, point $L$ is the midpoint of $C E$, and point $M$ is the midpoint of $B D$. The area of triangle $K L M$ is $\\frac{\\sqrt{3}}{5}$. Find $B D$.",
        "answer": "\\frac{2-\\sqrt{3}}{\\sqrt{5}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62705,
        "difficulty": null
      },
      {
        "id": "a81d9a62f62806f31751dbf4fadae95666a3628caf9db390a9897b44163512a1",
        "domain": "numina",
        "question": "Let's determine those five-digit numbers, which, when divided by the four-digit number obtained by omitting the middle digit, result in an integer.\n\n \n\nFind the five-digit numbers such that when divided by the four-digit number obtained by omitting the middle digit, the result is an integer.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169069,
        "difficulty": null
      },
      {
        "id": "9aebf9b1adb9a889230b5139a3094e950bc944d9fd0e104037bdafcaac47e907",
        "domain": "numina",
        "question": "Task 1. If to a two-digit number the digit 5 is added once at the beginning, and another time at the end, two different numbers are obtained whose difference is 252. What is that two-digit number?",
        "answer": "2783",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188508,
        "difficulty": null
      },
      {
        "id": "64b113504818c324ec314f0db1996b9b98f522ec5564620e701d027ce0e4e814",
        "domain": "numina",
        "question": "Let $E=\\{a, b, c, d\\}$ be a set. How many functions $f$ from $E$ to $E$ satisfy, for all $x$ in $E, f(f(f(x)))=x$?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 81188,
        "difficulty": null
      },
      {
        "id": "8d19cb7b77e488399d56261b14447a36d0548fc6a84bba809592957a38f45d1a",
        "domain": "numina",
        "question": "Problem 8.1. A square was cut into four equal rectangles, and from them, a large letter P was formed, as shown in the figure, with a perimeter of 56.\n![](https://cdn.mathpix.com/cropped/2024_05_06_e7f9c87b6a37ffba3564g-27.jpg?height=356&width=720&top_left_y=274&top_left_x=366)\n\nWhat is the perimeter of the original square",
        "answer": "32",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15579,
        "difficulty": null
      },
      {
        "id": "edbb638f116f2af8c61266fedf835220c0bc171abdb5b71c8c3c64c31a039142",
        "domain": "numina",
        "question": "3. Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence with the sum of the first $n$ terms denoted as $S_{n}$. If $S_{6}=26, a_{7}=2$, then the maximum value of $n S_{n}$ is $\\qquad$ .",
        "answer": "338",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118494,
        "difficulty": null
      },
      {
        "id": "5f3a638efbfe7ee7ce218737bf989f14ba82044859c9c59b16a16bf952632f72",
        "domain": "numina",
        "question": "Problem 9.4. A prime number $p$ is such that the number $p+25$ is the seventh power of a prime number. What can $p$ be? List all possible options.",
        "answer": "103",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16734,
        "difficulty": null
      },
      {
        "id": "24089efe8e23b453caacccb5a3cd91a89d7c1da184006afdbd11edc57e19fd10",
        "domain": "numina",
        "question": "For how many dice is the probability the greatest that exactly one six will appear when the dice are rolled simultaneously?",
        "answer": "56",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 169926,
        "difficulty": null
      },
      {
        "id": "4b2d986f96a94844eaa6a16ebb8ad64d539e89e46b4c4246aeee3befb23cf407",
        "domain": "numina",
        "question": "In the plane, 2013 red points and 2014 blue points are marked so that no three of the marked points are collinear. One needs to draw \\( k \\) lines not passing through the marked points and dividing the plane into several regions. The goal is to do it in such a way that no region contains points of both colors. Find the minimal value of \\( k \\) such that the goal is attainable for every possible configuration of 4027 points. (Australia)",
        "answer": "2013",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 196153,
        "difficulty": null
      },
      {
        "id": "0f22e99ae024658ad13e7c43089cdeb9426dce96ae9a3f289ba0606905df543b",
        "domain": "numina",
        "question": "Problem 4.5. Hooligan Dima made a construction in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the figure.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire construction to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
        "answer": "65",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17058,
        "difficulty": null
      },
      {
        "id": "6887f00f950d8d279d22df24ce473372c0ed82129c3619d85e59e13add6155cf",
        "domain": "numina",
        "question": "4. Person A and Person B agreed to have dinner at a restaurant. Since this restaurant is very popular, after A arrived, they took a waiting number and waited for B. B arrived at the restaurant a while later but didn't see A, so they also took a waiting number. While waiting, B saw A, and both of them showed their waiting numbers. They found that the digits of the two numbers were reverse two-digit numbers, and the sum of the digits (for example, the sum of the digits of the number 23 is $2+3=5$) of both numbers was 8, and B's number was 18 more than A's. What is A's number? $\\qquad$",
        "answer": "35",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 120865,
        "difficulty": null
      },
      {
        "id": "deec1cb05886bd2d55893b3f4c5eeb21b4af26e11cb957fb9e3c6a8adf436342",
        "domain": "numina",
        "question": "43. Without using any unit of length or other auxiliary means, how can you cut exactly half a meter from a piece of fabric that is $\\frac{8}{15}$ m?",
        "answer": "\\frac{1}{2}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 39533,
        "difficulty": null
      },
      {
        "id": "56ef4424009dbd7dc7e57e5f0d5d85edef756cd796f736bfe55025f46f90514d",
        "domain": "numina",
        "question": "Determine the number of all positive ten-digit integers with the following properties:\n\n- The number contains each of the digits 0, 1, 2, ..., 8, and 9 exactly once.\n- Each digit, except for the 9, has a neighboring digit that is greater than it.\n\n(Note. For example, in the number 1230, the digits 1 and 3 are the neighboring digits of 2, and 2 and 0 are the neighboring digits of 3. The digits 1 and 0 have only one neighboring digit.)\n\n(Karl Czakler)\n\nAnswer. There are 256 numbers with the required properties.",
        "answer": "256",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 190971,
        "difficulty": null
      },
      {
        "id": "42d15e9d3ddcecaa769a97e313424e9d5d2fa5c1d7848df567f8581d910c332f",
        "domain": "numina",
        "question": "1. Calculate the value of the numerical expression: $36+64 \\cdot 17-(502-352: 8+511)$.",
        "answer": "155",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86616,
        "difficulty": null
      },
      {
        "id": "1091fcb05ad3e4d3341f5a6c0a83f523507319e508ff2f65bc3694ba5f42e8ca",
        "domain": "numina",
        "question": "5. Calculate $\\sqrt{n}+\\sqrt{n+524}$, given that this number is rational and that $n$ is a natural number.",
        "answer": "262",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6120,
        "difficulty": null
      },
      {
        "id": "a8197bbf742984f849e521e900b762e39a68a4c620bbdc0a5c3dd6cf7560270e",
        "domain": "numina",
        "question": "5. (4 points) Let two sequences of numbers $\\left(x_{0}, x_{1}, \\ldots, x_{2009}\\right)$, $\\left(y_{0}, y_{1}, \\ldots, y_{2009}\\right)$ be constructed according to the following rules:\na) $x_{0}=12, x_{1}=\\frac{1}{3}, y_{0}=4, y_{1}=\\frac{1}{18}$,\n\nb) $x_{i+1}=x_{i-1}+4 x_{i}$ and $y_{i+1}=y_{i-1}-4 y_{i}$ for $i=1, \\ldots, 2008$.\n\nCalculate the value of $x_{2009} y_{2008}+x_{2008} y_{2009}$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8619,
        "difficulty": null
      },
      {
        "id": "5c743afc3bd9d336b3ffd8fbb19918569f0189be2a72a04b0d8b333b8a71e16f",
        "domain": "numina",
        "question": "4. let the positive divisors of the natural number $n$ be $1=d_{1}<d_{2}<\\ldots<d_{k}=n$. Determine all $n$ for which the following applies\n\n$$\n2 n=d_{5}^{2}+d_{6}^{2}-1\n$$\n\n## Solution",
        "answer": "272",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 91270,
        "difficulty": null
      },
      {
        "id": "addad5b9368aca437d020463bdb86868cff317d25b730b346a2230039eb0a79f",
        "domain": "numina",
        "question": "1. The clock shows 00:00, at which the hour and minute hands of the clock coincide. Considering this coincidence as number 0, determine after what interval of time (in minutes) they will coincide for the 19th time. If necessary, round the answer to the hundredths place according to rounding rules.",
        "answer": "1243.64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10306,
        "difficulty": null
      },
      {
        "id": "38cfe928beb80af2873de6650e2e2021d3a099d298032b0a5f8ddcd06bd0dc3a",
        "domain": "numina",
        "question": "Task 2. Determine all positive integers $n$ for which there exist positive integers $a_{1}, a_{2}, \\ldots, a_{n}$ such that\n\n$$\na_{1}+2 a_{2}+3 a_{3}+\\ldots+n a_{n}=6 n\n$$\n\nand\n\n$$\n\\frac{1}{a_{1}}+\\frac{2}{a_{2}}+\\frac{3}{a_{3}}+\\ldots+\\frac{n}{a_{n}}=2+\\frac{1}{n}\n$$",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85404,
        "difficulty": null
      },
      {
        "id": "9ae31923423b68d4932ad6e61565cdf84e529eed56120244038747fe9e31aec6",
        "domain": "numina",
        "question": "(5) In a regular tetrahedron $ABCD$, $AO \\perp$ plane $BCD$, with the foot of the perpendicular being $O$. Let $M$ be a point on the line segment $AO$ such that $\\angle BMC=90^{\\circ}$. Then $\\frac{AM}{MO}=$ $\\qquad$ .",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 99755,
        "difficulty": null
      },
      {
        "id": "eaa47a7badb4fb6b48f956752ff40ac3edeb0899c5eaf6ee071d2032daf7c9af",
        "domain": "numina",
        "question": "Example 29 (2003 National High School Mathematics Competition Question) Let the three sides of a triangle be integers $l, m, n$, and $l>m>n$. It is known that $\\left\\{\\frac{3^{l}}{10^{4}}\\right\\}=\\left\\{\\frac{3^{m}}{10^{4}}\\right\\}=\\left\\{\\frac{3^{n}}{10^{4}}\\right\\}$, where $\\{x\\}=x-[x]$, and $[x]$ represents the greatest integer not exceeding $x$. Find the minimum perimeter of such a triangle.",
        "answer": "3003",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103763,
        "difficulty": null
      },
      {
        "id": "8a7567a02fc513f6ab3254431309a7a044615a1863b5c0e938a9e537660ad806",
        "domain": "numina",
        "question": "3. In the sequence $\\left\\{a_{n}\\right\\}$, for $1 \\leqslant n \\leqslant 5$, we have $a_{n}=n^{2}$; and for all positive integers $n$, we have $a_{n+5}+a_{n+1}=a_{n+4}+a_{n}$. Then $a_{2023}=$ $\\qquad$ .",
        "answer": "17",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117621,
        "difficulty": null
      },
      {
        "id": "2a55267960a2f802c59b2f112093b6c89ce87663ab4ce78f30b22db680850241",
        "domain": "numina",
        "question": "Example 1. Find the volume of the body $\\Omega$, bounded by the surfaces\n\n$$\nx=17 \\sqrt{2 y}, \\quad x=2 \\sqrt{2 y}, \\quad z=\\frac{1}{2}-y, \\quad z=0\n$$",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31570,
        "difficulty": null
      },
      {
        "id": "f179e7e87e085a41a7b3b38880c044cbef452f4d50e1d1cd171a4a117059966d",
        "domain": "numina",
        "question": "Let $A$ be the following.\n\nA numerical sequence is defined by the conditions: $a_{1}=1, a_{n+1}=a_{n}+\\left[\\sqrt{a_{n}}\\right]$.\n\nHow many perfect squares occur among the first terms of this sequence, not exceeding\n\n1000000?",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 67676,
        "difficulty": null
      },
      {
        "id": "7ae47fd9acbdadb9fab33a376d3f4593e85e90ee251721bb3d73f3fcaf10e9d9",
        "domain": "numina",
        "question": "1. Let the set $P_{n}=\\{1,2, \\cdots, n\\}\\left(n \\in \\mathbf{Z}_{+}\\right)$. Denote $f(n)$ as the number of sets $A$ that satisfy the following conditions:\n(1) $A \\subseteq P_{n}, \\bar{A}=P_{n} \\backslash A$;\n(2) If $x \\in A$, then $2 x \\notin A$;\n(3) If $x \\in \\bar{A}$, then $2 x \\notin \\bar{A}$.\nThen $f(2018)=$ $\\qquad$",
        "answer": "2^{1009}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 142657,
        "difficulty": null
      },
      {
        "id": "19e48b70cad026cd6f65d1e010565ef1e4aa94f42b2b4429319310ff0f2ef0d4",
        "domain": "numina",
        "question": "5. Let $[x]$ denote the greatest integer not exceeding the real number $x$. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies\n$$\na_{1}=a_{2}=\\frac{1}{2}, a_{n+1}=2 a_{n}+a_{n-1}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen $\\left[\\sum_{k=2}^{2016} \\frac{1}{a_{k-1} a_{k+1}}\\right]=$ $\\qquad$ .",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112874,
        "difficulty": null
      },
      {
        "id": "08b7205fbcf76040e3e517473a01aa95e5b13230c81f369b08872edf4dce8cc6",
        "domain": "numina",
        "question": "2. In a triangle, the length of one median is half longer than the length of the side to which it is drawn. Find the angle between the other two medians.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187630,
        "difficulty": null
      },
      {
        "id": "f1a30a1b0a5127f07ef255bb0a48ef568478d351d5d6f238cc181f8a84f26241",
        "domain": "numina",
        "question": "38. How many six-digit numbers are there in which only the middle two digits are the same?",
        "answer": "27216",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44859,
        "difficulty": null
      },
      {
        "id": "6b0532da40326624ac66965aa393e3f0ee0a188bcae97d3d466902be241a2651",
        "domain": "numina",
        "question": "75. Исследование пустыни. Девять участников экспедиции (каждый на автомашине) встречаются на восточной окраине пустыни. Они хотят исследовать ее внутренние районы, двигаясь все время на запад. Каждому автомобилю полного бака (содержащего 1 галлон бензина) хватает на 40 миль пути. Кроме того, он может взять с собой еще 9 канистр бензина по галлону каждая (но не больше). Целые канистры можно передавать с одного автомобиля на другой. На какое максимальное расстояние исследователи могут проникнуть в пустыню, не создавая складов топлива, необходимого для возвращения назад",
        "answer": "360",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 40991,
        "difficulty": null
      },
      {
        "id": "21ccc2cdc49401e83be899252e0c62082385d0808963dace05e56eda7b8f6d00",
        "domain": "numina",
        "question": "## 22. General Cleaning\n\nA solid joint-stock company occupies three floors in a tower, which has the shape of a parallelepiped and is located in Paris in the La Défense district: the 13th and 14th floors, where the company's offices are located, and the 25th floor, where its board of directors is situated.\n\nTo clean these premises, a cleaning crew is called in, who work for four hours in the offices. Then the crew splits into two parts: half goes to clean the 25th floor, while the other half continues to clean the 13th and 14th floors. After another four hours, the crew stops working: the two lower floors are already completely cleaned, while the 25th floor is scheduled to receive another cleaner the next day, who will complete the work in 8 hours.\n\nAssuming that the time required for cleaning is proportional only to the area of the cleaned premises, how many cleaners were in the crew called by the company?",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 42201,
        "difficulty": null
      },
      {
        "id": "f85cef34a294929ae19cd1fe6665cc7a051680435a96dd3bcb77d63db896a25f",
        "domain": "numina",
        "question": "2. Find the value of the expression\n\n$$\n2^{2}+4^{2}+6^{2}+\\ldots+2018^{2}+2020^{2}-1^{2}-3^{2}-5^{2}-\\ldots-2017^{2}-2019^{2}\n$$",
        "answer": "2041210",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13735,
        "difficulty": null
      },
      {
        "id": "208bfb4007f55ef54d2d430f34b59943500cdce0a5b54b6889ed9854136ba70d",
        "domain": "numina",
        "question": "31. Let $x, y, z$ be real numbers such that\n$$\nx+y+z=1 \\quad \\text { and } \\quad x^{2}+y^{2}+z^{2}=1\n$$\n\nLet $m$ denote the minimum value of $x^{3}+y^{3}+z^{3}$. Find $9 m$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160943,
        "difficulty": null
      },
      {
        "id": "477f5282fc0bef55976aaf2d5d715fdc3d0cf5cdccfae52c9cc4a50e6616ffc5",
        "domain": "numina",
        "question": "Let $p$ and $q$ be different prime numbers. In how many ways can the fraction $1 / p q$ be decomposed into the sum of the reciprocals of 2 different natural numbers?",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 164346,
        "difficulty": null
      },
      {
        "id": "bc1db030b96aaf5e78e7a43d5377bffd1d4510c871d2d1242aedfd9889f75cc9",
        "domain": "numina",
        "question": "2. On the written math test, the first problem was not solved by $12 \\%$ of the students, $32 \\%$ solved it partially, and the remaining 14 students solved it correctly. How many students were in this class?",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188103,
        "difficulty": null
      },
      {
        "id": "3735c732a19ba9cac7179ff0f4070a7d6c4119bee2fac1bfbc051ae48c997966",
        "domain": "numina",
        "question": "3 [ [sphere, circumscribed around a pyramid]\n\nA regular quadrilateral pyramid is inscribed in a sphere. The radius of the sphere is 1. The plane angle at the vertex of the pyramid is $45^{\\circ}$. Find the lateral surface area of the pyramid.\n\n#",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66177,
        "difficulty": null
      },
      {
        "id": "32ea60a8c250e5029b6d1c7022aac6d928872c31c9f593f5a52be3c924351eb4",
        "domain": "numina",
        "question": "47. As shown in the figure, in rectangle $A B C D$, $M$ is the midpoint of $A B$, $A N$ is an arc of a circle with center $D$ and radius 3, $K N$ is an arc of a circle with center $C$, and the area of the shaded region is $10-\\frac{5 \\pi}{2}$. What is the length of the radius of $K N$?",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 134527,
        "difficulty": null
      },
      {
        "id": "5b2fd1e85ef608ce6744b86c4a97d1d3526fa08033396580e9641d73bb2624a2",
        "domain": "numina",
        "question": "I2.1 Given that functions $f(x)=x^{2}+r x+s$ and $g(x)=x^{2}-9 x+6$ have the properties that the sum of roots of $f(x)$ is the product of the roots of $g(x)$, and the product of roots of $f(x)$ is the sum of roots of $g(x)$. If $f(x)$ attains its minimum at $x=a$, find the value of $a$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153347,
        "difficulty": null
      },
      {
        "id": "98517def128576318ff5b6b09bcf81f9275a56f01a3aae415984dc07d1966a15",
        "domain": "numina",
        "question": "Example 2.41. $I=\\int_{0}^{1} \\ln x d x$.",
        "answer": "-1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30878,
        "difficulty": null
      },
      {
        "id": "d09f3056244eba8ea0607e641c3d19dc5eeee3bdb587992ffe9af7831f3e0949",
        "domain": "numina",
        "question": "$\\underline{\\text { Folklore }}$\n\nIn triangle $ABC$, the median drawn from vertex $A$ to side $BC$ is four times smaller than side $AB$ and forms an angle of $60^{\\circ}$ with it. Find angle $A$.",
        "answer": "150",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61447,
        "difficulty": null
      },
      {
        "id": "d66e4715d82e48c2215c45b7ddc547f693ef4ff9dc28533eae68a4b8498712c5",
        "domain": "numina",
        "question": "3. If for all positive numbers $x, y$, we have $\\sqrt{x}+\\sqrt{y} \\leq a \\sqrt{x+y}$, then the minimum value of the real number $a$ is",
        "answer": "\\sqrt{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 125271,
        "difficulty": null
      },
      {
        "id": "e13c4533f6e208ca1ce79b067094ca1a41f38e5fdfdab8113857975126470160",
        "domain": "numina",
        "question": "Let's determine those positive integers which are 14 times as large as the sum of their digits.",
        "answer": "126",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171188,
        "difficulty": null
      },
      {
        "id": "7486c42bff0903390bc7ff37b54207cd9b062fc02c3b1c2483944b0dba550e71",
        "domain": "numina",
        "question": "Find the largest four-digit number, all digits of which are different and which is divisible by 2, 5, 9, and 11.\n\n#",
        "answer": "8910",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 71234,
        "difficulty": null
      },
      {
        "id": "52a3c499d2a40e7e82196554a4de4ee87c6a8c8c2e7c845951cb0aa840a3fbc7",
        "domain": "numina",
        "question": "22. When proving the Pythagorean theorem, four congruent right triangles can be used to form a square (as shown in Figure 1), and we call such a figure a \"string figure.\"\nA similar method of constructing string figures is often used in rectangles. As shown in Figure 2, draw a line $EF$ through vertex $A$ of rectangle $ABCD$, and draw perpendiculars from points $B$ and $D$ to $EF$, with the feet of the perpendiculars being points $E$ and $F$, respectively, then $\\triangle BEA \\sim \\triangle AFD$.\nAs shown in Figure 3, in rectangle $ABCD$, $AD=5$, $AB=10$, construct congruent right triangles $ADE$ and $CBF$ with $AD$ and $BC$ as the hypotenuses, respectively. If $AE=CF=3$, then $EF^{2}=$ . $\\qquad$",
        "answer": "221",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124813,
        "difficulty": null
      },
      {
        "id": "e525176d2bdda35004103c7a69f4c67fac7aae9e7cd5b0079950eb0b755e48d0",
        "domain": "numina",
        "question": "7 chocolates are more expensive than 8 packs of cookies. What is more expensive - 8 chocolates or 9 packs of cookies?\n\n#",
        "answer": "8",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 71712,
        "difficulty": null
      },
      {
        "id": "981cbd4fe4bf32c71d9020bf7935dcc59d703959f881e70145b33267bf2f6e05",
        "domain": "numina",
        "question": "## Task 1 - 080621\n\nIn a 6th grade class, no student received a final grade of 5 in mathematics, every ninth student received a grade of 1, every third student received a grade of 2, and every sixth student received a grade of 4.\n\nThe number of students $n$ in this class is known to be: $20<n<40$.\n\nCalculate the number of students who received a final grade of 3!",
        "answer": "14",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 183230,
        "difficulty": null
      },
      {
        "id": "2182749d7248597fffe260615f944318d8a1e287c06ca23b3105799d79aaf756",
        "domain": "numina",
        "question": "4. The numbers $x$ and $y$ are such that the equalities $\\sin y + \\cos x = \\sin 3x$ and $\\sin 2y - \\sin 2x = \\cos 4x - \\cos 2x$ hold. What is the smallest value that the sum $\\cos y + \\sin x$ can take?",
        "answer": "-1-\\frac{\\sqrt{2+\\sqrt{2}}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3525,
        "difficulty": null
      },
      {
        "id": "815764897818d740b583fda492719a321f46654a5eaf2d1fb094800136f385fa",
        "domain": "numina",
        "question": "(10) There are 8 black, white, and yellow chopsticks each. Without looking, take out the chopsticks arbitrarily to ensure that there are at least two pairs of chopsticks of different colors. Then, at least $\\qquad$ chopsticks need to be taken out to achieve this.",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 126300,
        "difficulty": null
      },
      {
        "id": "bf77a01222a9f7b3d91c919218ceb614b76b3fd6977e10448c54affe8510a794",
        "domain": "numina",
        "question": "Seven people dine around a round table every Saturday evening. How many times is it possible to have dinner if everyone wants to have two new neighbors each time? What is the result for eight people?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 81359,
        "difficulty": null
      },
      {
        "id": "6b23c875ae365d8d667219c827391682964b90bf62fbb75d9dfa36ab44044dc2",
        "domain": "numina",
        "question": "## Task 2\n\nSubtract 7 from 19!\n\na) Write down the equation!\n\nb) Justify your solution with the basic problem!\n\na) $19-7=12$; b) $9-7=2$",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 186329,
        "difficulty": null
      },
      {
        "id": "b621ea811bce84a2e62294fe1d7e01683a2a8165ed7a28ffb366709bc843017c",
        "domain": "numina",
        "question": "2. (5 points) If twice a number minus half of the number equals 2013, then the number is $\\qquad$",
        "answer": "1342",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146283,
        "difficulty": null
      },
      {
        "id": "ed0a67370bf0a1b2d9ec1812580254548c4378728d3083ff4791149967969696",
        "domain": "numina",
        "question": "25. Find the number that when divided by 2 gives a remainder of 1, when divided by 3 gives a remainder of 2, when divided by 4 gives a remainder of 3, and when divided by 5 gives a remainder of 4.",
        "answer": "59",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35783,
        "difficulty": null
      },
      {
        "id": "50e37f56fa137e96d15a25398efaf82da129c5856287a3eeaf087c53297294e8",
        "domain": "numina",
        "question": "7. Variant 1. Roma decided to create his own multiplication table. The rows correspond to the numbers 12, $13,14, \\ldots, 60$, and the columns - to the numbers $15,16,17, \\ldots, 40$. In the cells of the table, he wrote the products of the pairs of numbers from the row and column. How many of these products will be even numbers?",
        "answer": "962",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12978,
        "difficulty": null
      },
      {
        "id": "9ca9bc21e6231c0942de3dedae46f04571979a79f3a2427fd1be659b6a5721c4",
        "domain": "numina",
        "question": "4. \"24 Points\" is a familiar math game to many people, the game process is as follows: arbitrarily draw 4 cards from 52 cards (excluding the jokers). Use the numbers on these 4 cards (from 1 to 13, where $A=1, J=11, Q=12, K=13$) to derive 24 using the four basic arithmetic operations. The person who finds the algorithm first wins. The game rules state that all 4 cards must be used, and each card can only be used once. For example, with $2,3,4, Q$, the algorithm $2 \\times Q \\times(4-3)$ can be used to get 24.\nIf in one game, you happen to draw $2,5, \\mathrm{~J}, \\mathrm{Q}$, then your algorithm is: $\\qquad$",
        "answer": "2\\times(11-5)+12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 120421,
        "difficulty": null
      },
      {
        "id": "6e487de7ead5cc995cbb948487e14aab76a80355ef3b22504df7edb3cc4d8ad2",
        "domain": "numina",
        "question": "66. A seven-digit number, from left to right, the sum of each pair of adjacent digits is 9, 7, 9, 2, 8, 11. This seven-digit number is $\\qquad$ .",
        "answer": "9072083",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 96676,
        "difficulty": null
      },
      {
        "id": "1517db3f1db775b5cfa542118e75701323ac0705b48d8a9625f83623a912f236",
        "domain": "numina",
        "question": "6. If $x^{3}-3 \\sqrt{2} x^{2}+6 x-2 \\sqrt{2}-8=0$, find the value of $x^{5}-41 x^{2}+2012$.\n(1 mark) If $x^{3}-3 \\sqrt{2} x^{2}+6 x-2 \\sqrt{2}-8=0$, find the value of $x^{5}-41 x^{2}+2012$.",
        "answer": "1998",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153655,
        "difficulty": null
      },
      {
        "id": "10e755e95a04c0df863201ce35333563cd5c6f2fbfdb6dc89bdcd8a6d697f16a",
        "domain": "numina",
        "question": "4. The given figure is an isosceles triangle $ABC$ with base $\\overline{AB}$ of length $65 \\, \\text{cm}$ and legs $\\overline{AC}$ and $\\overline{BC}$ of length $80 \\, \\text{cm}$. On the base $\\overline{AB}$, a point $D$ is chosen such that the perimeter of triangle $ADC$ is $173 \\, \\text{cm}$, and the perimeter of triangle $DBC$ is $220 \\, \\text{cm}$. What are the lengths of segments $\\overline{CD}, \\overline{AD}$, and $\\overline{DB}$?",
        "answer": "84",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86581,
        "difficulty": null
      },
      {
        "id": "91c0dbcc32b023faa274e0400e1231265554f3e348d0a6c587caeeb9fcd4dbd4",
        "domain": "numina",
        "question": "What can be the maximum area of a triangle if the length of any of its sides is not greater than 2?",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171560,
        "difficulty": null
      },
      {
        "id": "b11ff4dd8173ec304f9d2b554eccb7561490e6a257f073319d067bfb2fbde198",
        "domain": "numina",
        "question": "Example 2 As shown in Figure 1-10, 5 beads are strung together to form a necklace. Now, there are 3 colors of beads available for use. How many different necklaces can be made?",
        "answer": "39",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 115908,
        "difficulty": null
      },
      {
        "id": "c0cc1641a3787d3b9b7808c9a295b87995b51b8cfec5cff743e66d7cb80e1504",
        "domain": "numina",
        "question": "Problem 3. We are given a combination lock consisting of 6 rotating discs. Each disc consists of digits $0,1,2, \\ldots, 9$, in that order (after digit 9 comes 0 ). Lock is opened by exactly one combination. A move consists of turning one of the discs one digit in any direction and the lock opens instantly if the current combination is correct. Discs are initially put in the position 000000, and we know that this combination is not correct.\n\na) What is the least number of moves necessary to ensure that we have found the correct combination?\n\nb) What is the least number of moves necessary to ensure that we have found the correct combination, if we know that none of the combinations $000000,111111,222222, \\ldots, 999999$ is correct?\n\n(Ognjen Stipetić, Grgur Valentić)",
        "answer": "10^{6}-2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 189435,
        "difficulty": null
      },
      {
        "id": "6e2b8d3878b0065947be12c8d0aecb6356f11086b1d17fc34d5082e64fbb6db3",
        "domain": "numina",
        "question": "[ Number of divisors and sum of divisors of a number ] equations in integers $\\quad]$\n\nFind a natural number $n$, knowing that it has two prime divisors and satisfies the conditions $\\tau(n)=6$, $\\sigma(n)=28$.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69800,
        "difficulty": null
      },
      {
        "id": "d4252273fad29da38d658c9585dcee469d5dd359ed36a0c8929331d27b946072",
        "domain": "numina",
        "question": "2. Calculate $1+12+123+1234+12345+123456+1234567+12345678+123456789$, the hundreds digit of the sum is . $\\qquad$",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124124,
        "difficulty": null
      },
      {
        "id": "6366d72d9c02f2cb6c1597aab1780d5834ad28168d5305936bfe19be3cbbb11b",
        "domain": "numina",
        "question": "5. The lengths of the sides of an isosceles triangle are expressed as natural numbers (in centimeters). The product of the lengths of all sides of this triangle is 2016. What is the perimeter of this triangle?\n\n## Tasks worth 10 points:",
        "answer": "38",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86992,
        "difficulty": null
      },
      {
        "id": "41b005cbbe144a357dbc2979db421801d140117aad5e4e485487c3ea38c18152",
        "domain": "numina",
        "question": "1.1 $\\frac{\\frac{1}{a}-\\frac{1}{b+c}}{\\frac{1}{a}+\\frac{1}{b+c}}\\left(1+\\frac{b^{2}+c^{2}-a^{2}}{2 b c}\\right): \\frac{a-b-c}{a b c}$;\n\n$$\na=0.02, b=-11.05, c=1.07\n$$\n\n$1.2\\left(\\frac{1}{t^{2}+3 t+2}+\\frac{2 t}{t^{2}+4 t+3}+\\frac{1}{t^{2}+5 t+6}\\right)^{2} \\frac{(t-3)^{2}+12 t}{2}$.",
        "answer": "0.1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49456,
        "difficulty": null
      },
      {
        "id": "a5016964fd48f0dc2d383636111cd82dfb2ad7eaa5eec82b05c6d169f823a33d",
        "domain": "numina",
        "question": "14. Given that the sum of the sines of any 5 interior angles and the cosines of the remaining 4 interior angles of a convex nonagon equals a constant value $\\lambda$. If one of the nine interior angles is $120^{\\circ}$, find the value of the constant $\\lambda$.",
        "answer": "1-\\frac{\\sqrt{3}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 106112,
        "difficulty": null
      },
      {
        "id": "b1b359e2197a21df4c5616ef22e9c47595dff1afb1877397328dfc8cef036be0",
        "domain": "numina",
        "question": "4. Given that $n$ is a positive integer, if $\\frac{n^{2}+3 n-10}{n^{2}+6 n-16}$ is a reduced fraction, then the value of this fraction is $\\qquad$ .",
        "answer": "\\frac{8}{11}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130819,
        "difficulty": null
      },
      {
        "id": "83eaf2cd078f9ad7d6e71dab51a757c61ecfd18e7dd7161fe36a06ef161abe9c",
        "domain": "numina",
        "question": "Problem 5.8. Inside a large triangle with a perimeter of 120, several segments were drawn, dividing it into nine smaller triangles, as shown in the figure. It turned out that the perimeters of all nine small triangles are equal to each other. What can they be equal to? List all possible options.\n\nThe perimeter of a figure is the sum of the lengths of all its sides.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5b323dccc142e96a9907g-12.jpg?height=256&width=323&top_left_y=1257&top_left_x=565)",
        "answer": "40",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17920,
        "difficulty": null
      },
      {
        "id": "21e11f283ff9ba06b0dd73b3a722ea5e8f2fdc2d3221ffdc4bc4a7c6ae6ed42a",
        "domain": "numina",
        "question": "On the diagonal $B D$ of the rectangular trapezoid $A B C D\\left(\\angle D=90^{\\circ}, B C \\| A D\\right)$, a point $Q$ is taken such that $B Q: Q D=1: 3$. A circle with center at point $Q$ touches the line $A D$ and intersects the line $B C$ at points $P$ and $M$. Find the length of the side $A B$, if $B C=9, A D=8, P M=4$.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63703,
        "difficulty": null
      },
      {
        "id": "6fd3fc0e746738b542a6369408bc0afb7b0cb1f3ab4daaaf3a67179a6242659b",
        "domain": "numina",
        "question": "5. Five pairs of twin siblings participate in the activities of $k$ groups. If the rules are: (1) Twin siblings are not in the same group; (2) Any two people who are not twins have participated in exactly one group together; (3) One person participates in only two groups. Then the minimum value of $k$ is $\\qquad$ .",
        "answer": "14",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 105371,
        "difficulty": null
      },
      {
        "id": "7456e4465ec0f07c17c7e0b02f4e45d04e61ab056d2d0a4a957e900a7d5e576e",
        "domain": "numina",
        "question": "5. You and I play the following game on an $8 \\times 8$ square grid of boxes: Initially, every box is empty. On your turn, you choose an empty box and draw an $X$ in it; if any of the four adjacent boxes are empty, you mark them with an $X$ as well. (Two boxes are adjacent if they share an edge.) We alternate turns, with you moving first, and whoever draws the last $X$ wins. How many choices do you have for a first move that will enable you to guarantee a win no matter how I play?",
        "answer": "0",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 152317,
        "difficulty": null
      },
      {
        "id": "de622823d14ceb0d5771d0c55dfd9650d265966a5ccc415c086c601af90d9308",
        "domain": "numina",
        "question": "2. If $\\sin x+\\cos x=\\frac{\\sqrt{2}}{2}$, then $\\sin ^{3} x+\\cos ^{3} x=$ $\\qquad$",
        "answer": "\\frac{5\\sqrt{2}}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99875,
        "difficulty": null
      },
      {
        "id": "1e724001573041337c473ca9a8cf0bb962acc5147574841eb18964d5e2059dd5",
        "domain": "numina",
        "question": "I5.3 It is known that $b^{16}-1$ has four distinct prime factors, determine the largest one, denoted by $c$",
        "answer": "257",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152682,
        "difficulty": null
      },
      {
        "id": "23dc3653602dfc4c4bd02ea4910ba8cc0219af0f663f30e75b0684194c599ab7",
        "domain": "numina",
        "question": "B3 In the figure, two circles with radius 6 are drawn. The centers of the circles are also 6 units apart.\n\nDetermine the area of the shaded region, which is bounded by two circular arcs and a common tangent.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2582a2b087f6953c1edag-2.jpg?height=363&width=537&top_left_y=721&top_left_x=1316)",
        "answer": "36-9\\sqrt{3}-6\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 85876,
        "difficulty": null
      },
      {
        "id": "3fce18e9e6e8e8ca5407edd3720ea19147a604d38bbe7e22107d7b0c64e83008",
        "domain": "numina",
        "question": "A quantity of grey paint has a mass of $12 \\mathrm{~kg}$. The grey paint is a mixture of black paint and white paint, of which $80 \\%$ by mass is white paint. More white paint is added until the mixture is $90 \\%$ white paint by mass. What is the resulting total mass of the paint, in $\\mathrm{kg}$ ?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194274,
        "difficulty": null
      },
      {
        "id": "6cf70613c2553ff98f8655d9aab734a99b77d5f9d106bc51d5d2439e88df6133",
        "domain": "numina",
        "question": "1. Call a positive integer a hussel number if:\n\n(1) All digits are not equal to 0.\n\n(2) The number is divisible by 11.\n\n(3) The number is divisible by 12. If you rearrange the digits in any other random order, you always get a number that is divisible by 12.\n\nHow many 5-digit hussel numbers are there?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85773,
        "difficulty": null
      },
      {
        "id": "beea4329d05dedd899d3cd77fa62fd116c3cc0e1337cf368b3037544a1746c18",
        "domain": "numina",
        "question": "5. In trapezoid $A B C D$ with bases $A D / / B C$, the diagonals intersect at point $E$. It is known that the areas $S(\\triangle A D E)=12$ and $S(\\triangle B C E)=3$. Find the area of the trapezoid.",
        "answer": "27",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5341,
        "difficulty": null
      },
      {
        "id": "fbb0b7b4b9360040e7571d0fae370206ee41db000c0a718623c5301dba355551",
        "domain": "numina",
        "question": "1. In a contest, $\\mathbf{30}$ problems are given. For each correct answer, 5 points are awarded, and for each wrong answer, 3 points are deducted. How many correct answers did a student give who scored 118 points?",
        "answer": "26",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 180170,
        "difficulty": null
      },
      {
        "id": "5ae8aa091ae4f90711a08b0638cea9968541970161756ddc16cae336be4d27f2",
        "domain": "numina",
        "question": "8. Let $a, b$ be positive integers, and $a-b \\sqrt{3}=(2-\\sqrt{3})^{100}$, then the units digit of $a b$ is",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 108909,
        "difficulty": null
      },
      {
        "id": "51083c0c8beed09b63202e4d7b2233511cd57d2710e4fe9794affc739db14444",
        "domain": "numina",
        "question": "In the picture, you can see a pool with a long step on one of its walls. We started filling the empty pool with a constant flow and observed the water level. After\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_ac63ecbffe7e6b0bc0e7g-2.jpg?height=309&width=614&top_left_y=451&top_left_x=755)\n\n8 min, the water level had risen to a height of $20 \\mathrm{~cm}$ and had not yet reached the level of the step. After $23 \\mathrm{~min}$ of filling, the water level was at $55 \\mathrm{~cm}$ and the step had already been submerged for some time. After 35.5 min of filling, the pool was filled to a height of $80 \\mathrm{~cm}$. At what height $h$ from the bottom of the pool is the step located?\n\n(L. Šimünek)",
        "answer": "45",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76208,
        "difficulty": null
      },
      {
        "id": "11b485ddeee29772f53e54271cd85dea053b9cb5cd2b9298c467a026d2c33d7c",
        "domain": "numina",
        "question": "5. [5 points] Given the numbers $\\log _{\\sqrt{\\frac{x}{3}+3}}(6 x-14), \\log _{6 x-14}(x-1)^{2}, \\log _{x-1}\\left(\\frac{x}{3}+3\\right)$. For which $x$ are two of these numbers equal, and the third one less than them by 1?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3358,
        "difficulty": null
      },
      {
        "id": "3e56f5607336fb3a4a5af36c26feed088c10347ec333a23cde8cda8deb10f13e",
        "domain": "numina",
        "question": "Problem 2. In class 4-b, there were 24 students. Each student can choose whether to study the subjects of informatics and German. 15 students study informatics, 12 study German, and 8 students do not study either informatics or German. How many students from class 4-b study both informatics and German at the same time?",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 188691,
        "difficulty": null
      },
      {
        "id": "a8ab63824eb0bf2cb15b98257147e93b62c14ed2ac2164c57a7e2f96fca957be",
        "domain": "numina",
        "question": "Problem 2. $n$ mushroom pickers went to the forest and brought a total of 162 mushrooms (each brought at least one mushroom home). Boy Petya, upon learning this, declared: \"Some two of them must have brought the same number of mushrooms!\" For what smallest $n$ will Petya definitely be right? Don't forget to justify your answer.",
        "answer": "18",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9241,
        "difficulty": null
      },
      {
        "id": "7ea3cc69b90554292a1cf0aa5a8eade2ed64437991a18461cd4748eb8301e495",
        "domain": "numina",
        "question": "16. If a positive integer cannot be written as the difference of two square numbers, then the integer is called a \"cute\" integer. For example, 1,2 and 4 are the first three \"cute\" integers. Find the $2010^{\\text {th }}$ \"cute\" integer.\n(Note: A square number is the square of a positive integer. As an illustration, 1,4,9 and 16 are the first four square numbers.)",
        "answer": "8030",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160642,
        "difficulty": null
      },
      {
        "id": "c358b9bf76d37b59a90eab95492298621c10e47da554fb5db2a5b48aa443ea13",
        "domain": "numina",
        "question": "Example 4 Find the largest constant $k$, such that for $x, y, z \\in \\mathbf{R}^{+}$, we have\n$$\n\\frac{x}{\\sqrt{y+z}}+\\frac{y}{\\sqrt{z+x}}+\\frac{z}{\\sqrt{x+y}} \\leqslant k \\sqrt{x+y+z} .\n$$",
        "answer": "\\frac{\\sqrt{6}}{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 134290,
        "difficulty": null
      },
      {
        "id": "a82f7c1303215c22a8c28e8f00a332b50235e4673408bb105d058ff0b8da0cb2",
        "domain": "numina",
        "question": "Find all primes of the form $a^{2}-1$, with $a \\geqslant 2$ natural.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 79808,
        "difficulty": null
      },
      {
        "id": "838fc23f81297026f2b7c9d4a4526db81b70acdf625a741e3d9a2fdf00d7e22c",
        "domain": "numina",
        "question": "How should a rook move across the chessboard to visit each square exactly once and make the fewest number of turns?\n\n#",
        "answer": "14",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 60384,
        "difficulty": null
      },
      {
        "id": "840f1f4816d37480e5a67f2a46d503e5fc0ba14f389ff20170fb3c51489fba8e",
        "domain": "numina",
        "question": "20. Let $f: \\mathbb{Q} \\backslash\\{0,1\\} \\rightarrow \\mathbb{Q}$ be a function such that\n$$\nx^{2} f(x)+f\\left(\\frac{x-1}{x}\\right)=2 x^{2}\n$$\nfor all rational numbers $x \\neq 0,1$. Here $\\mathbb{Q}$ denotes the set of rational numbers. Find the value of $f\\left(\\frac{1}{2}\\right)$.\n20. Answer: 1\nSubstituting $x=\\frac{1}{2},-1,2$, we get\n$$\n\\begin{aligned}\n\\frac{1}{4} f\\left(\\frac{1}{2}\\right)+f(-1) & =\\frac{1}{2}, \\\\\nf\\left(\\frac{1}{2}\\right)+f(-1)+f(2) & =2, \\\\\n4 f(2) & =8 .\n\\end{aligned}\n$$\n\nSolving these equations, we get $f\\left(\\frac{1}{2}\\right)=1$. In fact the same method can be used to determine $f$. Letting $x=z, \\frac{z-1}{z}, \\frac{1}{1-z}$, we get\n$$\n\\begin{array}{l}\nz^{2} f(z)+f\\left(\\frac{z-1}{z}\\right)=2 z^{2}, \\\\\n\\left(\\frac{z-1}{z}\\right)^{2} f\\left(\\frac{z-1}{z}\\right)+f\\left(\\frac{1}{11-z}\\right)=2\\left(\\frac{z-1}{z}\\right)^{2}, \\\\\nf(z)+\\frac{1}{(1-z)^{2}} f\\left(\\frac{1}{1-z}\\right)=\\frac{z}{2(1-z)^{2}} . \\\\\n\\end{array}\n$$\n\nUsing Cramer's rule, we can solve this system of linear equations in the unknowns $f(z), f\\left(\\frac{z-1}{z}\\right), f\\left(\\frac{1}{1-z}\\right)$. We obtain\n$$\nf(z)=1+\\frac{1}{(1-z)^{2}}-\\frac{1}{z^{2}} .\n$$\n\nIndeed one can easily check that it satisfies the given functional equation.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160711,
        "difficulty": null
      },
      {
        "id": "c16c98c834b9ca12b5ec7140a9f569f6d8235f756c2a9bc1d09931c8b79216df",
        "domain": "numina",
        "question": "Three. (50 points)\nIn a square with a side length of 4, 2002 points are placed. How many pairs of points are there with a distance less than 3 (find the largest specific number)?",
        "answer": "500000",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 98248,
        "difficulty": null
      },
      {
        "id": "56dd77fa83d4dcbbeedc247b209aa59633748f74a23a241e03f0f4b746f76ddc",
        "domain": "numina",
        "question": "The image shows a payment order for electricity to a certain energy sales company.\n\n![](https://cdn.mathpix.com/cropped/2024_05_08_22585010b2534c7ab9ebg-38.jpg?height=646&width=1380&top_left_y=1163&top_left_x=339)\n\nEach month, the client provides the company with readings from a three-tariff meter installed in the apartment. The current month's readings are subtracted from the corresponding readings of the previous month to obtain the actual consumption for the month in each of the three tariff zones (peak, night, semi-peak). The consumption in each zone is then multiplied by the price per kilowatt-hour in that zone. By summing these amounts, the client gets the total payment for the month. In this example, the client will pay 660 rubles and 72 kopecks.\n\nThe company keeps track of electricity consumption and payments using the data provided by the client. The problem is that the company sometimes confuses the six numbers, rearranging them in any order, although it ensures that the current reading remains greater than the previous one. As a result, the company's calculation can be incorrect. If the company believes that the client owes more than they paid, it demands the difference.\n\nUsing the data from the illustrated receipt, find:\n\na) the maximum possible additional payment for March 2013 that the company may demand from the client;\n\nb) the expected value of the difference between the amount the company will charge and the amount the client has paid.\n\n#",
        "answer": "397.34",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55904,
        "difficulty": null
      },
      {
        "id": "a938328b4219afee103b6f3d7f31e231cbbfed543fcfba6630fef48eff2bf372",
        "domain": "numina",
        "question": "The side length of the equilateral triangle $ABC$ is $6 \\, \\text{cm}$. Starting from vertex $C$, a bug moves uniformly towards vertex $A$ at a speed of $4 \\, \\text{mm/s}$. At the same time, another bug starts from vertex $B$ and moves towards vertex $C$ at a speed of $3 \\, \\text{mm/s}$. How much time after their start will they be closest to each other, and what will this distance be?",
        "answer": "34.17",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170830,
        "difficulty": null
      },
      {
        "id": "4865655852a358257d07fc3daebf604377a83a48810110e9a5494f300df9dd32",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 2} \\frac{\\operatorname{arctg}\\left(x^{2}-2 x\\right)}{\\sin 3 \\pi x}$",
        "answer": "\\frac{2}{3\\pi}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47066,
        "difficulty": null
      },
      {
        "id": "b95e0a9d8e52ecca3b37e22a66733b07fae2c32f7bf0625898ebc4c6f8dca7d6",
        "domain": "numina",
        "question": "We want to draw a digit on each face of two identical cubes so that by appropriately rotating the cubes and placing them next to each other, we can read any day of the month, including single-digit days with a leading zero for uniformity. How many ways can we choose the digits to be drawn on each cube? (The relative positions of the six digits on one cube are not considered in the counting.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_02_9b382d5b0c4e2d739941g-1.jpg?height=353&width=591&top_left_y=398&top_left_x=756)",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 164869,
        "difficulty": null
      },
      {
        "id": "cf12bc2d178798dc95bd0f279bcefce85c9732def7e22b1b5936ae2407e7927a",
        "domain": "numina",
        "question": "[ Inequalities Problems. Case Analysis]\n\nLong, long ago, nine identical books cost 11 rubles and some kopecks, and thirteen such books cost 15 rubles and some kopecks.\n\nHow much did one book cost\n\n#",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70061,
        "difficulty": null
      },
      {
        "id": "62ddae734750f675e45f21a40780c7acb0d92d202994bb32f5d878ea01981b48",
        "domain": "numina",
        "question": "## Aufgabe 2\n\n$$\n\\begin{array}{ll}\n12+5-4 & 13-3+2 \\\\\n17+2+0 & 17-5-1\n\\end{array}\n$$",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186350,
        "difficulty": null
      },
      {
        "id": "0165d477077f22640209de2b4422d7c6835d498380289e7f9300dfe35d66d2d3",
        "domain": "numina",
        "question": "1. Let $A B C D$ be a convex quadrilateral with $A B=a, B C=b, C D=c$ and $D A=d$. Suppose\n\n$$\na^{2}+b^{2}+c^{2}+d^{2}=a b+b c+c d+d a\n$$\n\nand the area of $A B C D$ is 60 square units. If the length of one of the diagonals is 30 units, determine the length of the other diagonal.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 180465,
        "difficulty": null
      },
      {
        "id": "7c14d5e4fe0d7d542ff0d851a9328c5e6ec4f9855e5cd24cb05e4bed9a57eb62",
        "domain": "numina",
        "question": "## Problem 1\n\nAn obtuse angled triangle has integral sides and one acute angle is twice the other. Find the smallest possible perimeter.",
        "answer": "77",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187176,
        "difficulty": null
      },
      {
        "id": "5c0a0b69e20270ec4d0958bb6a68cda15903806458fa6657e612e6b760322b50",
        "domain": "numina",
        "question": "6. On the shore of a round island called Somewhere, there are 20 villages, each inhabited by 20 wrestlers. A tournament was held where each wrestler faced all wrestlers from all other villages. Village A is considered stronger than Village B if at least $k$ matches between wrestlers from these villages end with a victory for a wrestler from Village A. It turned out that each village is stronger than the next one in the clockwise direction. What is the maximum value that $k$ can have? (All wrestlers have different strengths, and the stronger wrestler always wins in a match.)",
        "answer": "290",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 25992,
        "difficulty": null
      },
      {
        "id": "ccf94eb6fecf5cda3e011d6505388fa6034c3846bbacbbebb3fd0173f968c662",
        "domain": "numina",
        "question": "Simplify the following expressions:\n\n$$\n[1+(\\operatorname{tg} x-\\operatorname{cotg} x) \\sin 2 x] \\cos x+\\cos 3 x\n$$\n\n$$\n\\frac{\\operatorname{tg}^{2} x-\\sin ^{2} x}{\\operatorname{cotg}^{2} x-\\cos ^{2} x}\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167996,
        "difficulty": null
      },
      {
        "id": "0f70c9d512a4eb62285e7120fa44913f9f308ad5e1b9b8b03ff249f499c234c3",
        "domain": "numina",
        "question": "9.5. On a checkerboard of size $8 \\times 8$, 8 checkerboard ships of size $1 \\times 3$ are placed such that no two cells occupied by different ships share any points. One shot is allowed to pierce all 8 cells of one row or one column. What is the minimum number of shots needed to guarantee hitting at least one ship?",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16557,
        "difficulty": null
      },
      {
        "id": "4e778c21361475cd19de92bfe71c6ad9af5eb86cb5c8e3cdca510eb295b40953",
        "domain": "numina",
        "question": "Jóska and Gábor every summer watched the license plates of 51 cars passing by each day, and if all four digits of the number were different, Jóska got 1 point, otherwise Gábor did. After 40 days, Gábor claimed that he won because he had more points on 21 days. However, Jóska claims that if they added up the results of all 40 days, he would have more points. - In a long game, who has a better chance of accumulating more points? Based on what assumptions did we answer the question?",
        "answer": "5040",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 167604,
        "difficulty": null
      },
      {
        "id": "3cd92bf3b51fa6560d25be902760fe7e8e918a41beb172096cb19002efd3fcd2",
        "domain": "numina",
        "question": "Let's find a perfect square consisting of 4 digits, knowing that the number formed by the first two digits exceeds the number formed by the last two digits by one.",
        "answer": "8281",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 175064,
        "difficulty": null
      },
      {
        "id": "169ce137b3cbf10023cc0d5a03648076f4882390967cf326aee246afd012d105",
        "domain": "numina",
        "question": "3. In all other cases - 0 points\n\n## *Important: the numerical assessment of the free area (solution) is not the only possible one, for example, the \"gap\" can be more than 10 m.\n\n## Assignment 2 (12 points)\n\nCrocodile Gena and Old Lady Shapoklyak entered into a futures contract, according to which Gena agreed to invest in the 1st project an amount equal to $p_{1}>0$ thousand rubles, and after a year receive $x_{1}$ thousand rubles such that\n\n$$\n4 x_{1}-3 p_{1}-44=0\n$$\n\nand at the same time Gena agreed to invest in the 2nd project an amount equal to $p_{2}>0$ thousand rubles, and after a year receive $x_{2}$ thousand rubles such that\n\n$$\np_{2}^{2}-12 p_{2}+x_{2}^{2}-8 x_{2}+43=0\n$$\n\nAccording to the terms of the contract, the parameters $p_{1}, x_{1}, p_{2}, x_{2}$ were chosen so that the value\n\n$$\nd=\\sqrt{\\left(x_{1}-x_{2}\\right)^{2}+\\left(p_{1}-p_{2}\\right)^{2}}\n$$\n\nwas minimized. Determine:\n\n1) the minimum value of the quantity $d$;\n2) is the deal profitable for Gena? Find the amount of profit or loss for Gena if he agrees to this deal.",
        "answer": "13080",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7603,
        "difficulty": null
      },
      {
        "id": "08d32ce92ac4c163d9a6725275eac7bc1dc78d2b9b35665712b80c04a78e0dd4",
        "domain": "numina",
        "question": "## Task A-4.2.\n\nExactly three interior angles of a convex polygon are obtuse. What is the maximum number of sides that this polygon can have?",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88008,
        "difficulty": null
      },
      {
        "id": "b700e96888bf994a9b8d23eb889585afcfd2f9ad9416889f5b9286870bcbda44",
        "domain": "numina",
        "question": "4. (3 points) The rules of a musical instrument competition stipulate: The preliminary round is divided into four rounds, and the average score of the four rounds must be no less than 96 to advance to the final. Xiao Guang's scores in the first three rounds are $95$, $97$, and $94$. Therefore, to advance to the final, his score in the fourth round must be at least $\\qquad$ points.",
        "answer": "98",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120219,
        "difficulty": null
      },
      {
        "id": "144ca843a07556ae951ab0f3adceb03aad0fde3e50abe329cb8c628c19e2aeed",
        "domain": "numina",
        "question": "7.3.1. (12 points) Find the sum of all real roots of the equation\n\n$$\n\\sin \\left(\\pi\\left(x^{2}-x+1\\right)\\right)=\\sin (\\pi(x-1))\n$$\n\nbelonging to the interval $[0 ; 2]$.",
        "answer": "4.73",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6520,
        "difficulty": null
      },
      {
        "id": "d37efb4ba4c3034a117fb0a5a9da5079cb5d23f7a332e13584451e3e3f12744f",
        "domain": "numina",
        "question": "I3.1 Given that $\\frac{2 x-3}{x^{2}-x}=\\frac{A}{x-1}+\\frac{B}{x}$, where $A$ and $B$ are constants. If $S=A^{2}+B^{2}$, find the value of $S$",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153081,
        "difficulty": null
      },
      {
        "id": "895a9744c7460d41606f0d082ca3c23a6ce6edaab4b97b99d5da305de814d13e",
        "domain": "numina",
        "question": "(5) The minimum value of the function $f(x)=\\sqrt{x^{2}+9}+\\sqrt{x^{2}-8 x+17}$ is",
        "answer": "4\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101343,
        "difficulty": null
      },
      {
        "id": "e760dde9fbfc8b5952cad30111f17e666d23a0484e68e9f6aa2ae5d72ef8210b",
        "domain": "numina",
        "question": "# Task 3. (12 points)\n\nEgorov decided to open a savings account to buy a car worth 900000 rubles. The initial deposit amount is 300000 rubles. After one month and subsequently every month, Egorov plans to top up his account by 15000 rubles. The bank accrues interest monthly at an annual rate of $12 \\%$. Interest accrued for the month is credited to the account, and in the following month, interest is also accrued on this interest. What is the minimum number of months required for the account to have a sufficient amount to buy the car?",
        "answer": "29",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 22668,
        "difficulty": null
      },
      {
        "id": "f01f199a2c3bb7058f18c0950748acb1a81ec390a1d0455956751daa90f81ef5",
        "domain": "numina",
        "question": "3. In the sum $(1-2-3+4)+(5-6-7+8)+\\ldots+(2009-2010-2011+2012)+$ (2013-2014-2015), all numbers that end in 0 were crossed out. Find the value of the resulting expression.",
        "answer": "-3026",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15524,
        "difficulty": null
      },
      {
        "id": "cbb5758c480a61286bda301f10d26e0a4735c81698e641d02455a4a72df6b1c6",
        "domain": "numina",
        "question": "1. Determine the number of pairs of positive integers $(a, b)$, with $a \\leq b$, for which $\\operatorname{lcm}(a, b)=2004$.\n\n$\\operatorname{lcm}(a, b)$ denotes the least common multiple of $a$ and $b$.\n\nExample: $\\operatorname{lcm}(18,24)=72$.",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85759,
        "difficulty": null
      },
      {
        "id": "24e699543ce9aac90edbfd41bc51deee07c1ac87bd17d334463b12ac900c2bcf",
        "domain": "numina",
        "question": "20. A convex quadrilateral paper sheet has $n$ points inside it. Together with the 4 vertices of the quadrilateral, there are $n+4$ points, and no three points are collinear. Using these points as vertices, the quadrilateral is cut into several triangular paper pieces. The maximum number of triangles that can be cut is 1992, then $n=$",
        "answer": "9955",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92009,
        "difficulty": null
      },
      {
        "id": "279251ef6e90a3144b19617f3afff5edaf5ba2d61369752b57679e6b0b2e17f4",
        "domain": "numina",
        "question": "A sequence of numbers $1, 4, 7, 10, \\cdots, 697, 700$ follows the rule: the first number is 1, and each subsequent number is 3 more than the previous one, up to 700. If all these numbers are multiplied together, find the number of trailing zeros in the resulting product (for example, the number of trailing zeros in 12003000 is 3).",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113739,
        "difficulty": null
      },
      {
        "id": "110e5b95c6651c12622e264e7b7ac64ab4f1a15e218420f2ce47a8886ff224e1",
        "domain": "numina",
        "question": "6. A motorcyclist traveled 980 km over three days. In the first two days, he traveled 725 km, and on the second day, he traveled 123 km more than on the third day. How many kilometers did he travel each of these three days?",
        "answer": "347",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42697,
        "difficulty": null
      },
      {
        "id": "571f47a3e621c5ea719b2ca8a6897826c9a84fb222e4e3ad3256da8c28968a9d",
        "domain": "numina",
        "question": "$7.73 \\sqrt{\\log _{0.04} x+1}+\\sqrt{\\log _{0.2} x+3}=1$.\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly.\n\n$7.73 \\sqrt{\\log _{0.04} x+1}+\\sqrt{\\log _{0.2} x+3}=1$.",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49801,
        "difficulty": null
      },
      {
        "id": "a6026a60f33097676fe398cec8d1aa6032ad3e200ae9ee4eac79604231b4aae1",
        "domain": "numina",
        "question": "Problem 1. Children went to the forest to pick mushrooms. If Anya gives half of her mushrooms to Vitya, all the children will have the same number of mushrooms, and if instead Anya gives all her mushrooms to Sasha, Sasha will have as many mushrooms as all the others combined. How many children went mushroom picking?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22791,
        "difficulty": null
      },
      {
        "id": "dd97471749656f50187345d8c264e340b6e1cb3bb410fb8fc19e7311f7164883",
        "domain": "numina",
        "question": "14. (15 points) The distance between location A and location B is 360 kilometers. A truck loaded with 6 boxes of medicine is driving from location A to location B, while at the same time, a motorcycle starts from location B and heads towards the truck. The truck's speed is 40 kilometers/hour, and the motorcycle's speed is 80 kilometers/hour. When the motorcycle meets the truck, it unloads 2 boxes of medicine and returns to location B. It then immediately turns around... Each time the motorcycle meets the truck, it unloads 2 boxes of medicine and returns to location B. How long will it take to transport all 6 boxes of medicine to location B? How far will the motorcycle have traveled in total by then? (Neglect the time for loading and unloading the medicine.)",
        "answer": "8\\frac{2}{3}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 144670,
        "difficulty": null
      },
      {
        "id": "177d584d1f005d9d97a1626201314cc07b8be67fcccc4f159a319b8ef5780fc7",
        "domain": "numina",
        "question": "11. (5 points) As shown in the figure, the \"L\" shaped paper piece with a perimeter of 52 centimeters can be divided into two identical rectangles along the dotted line. If the longest side is 16 centimeters, then the area of the \"L\" shaped paper piece is $\\qquad$ square centimeters.",
        "answer": "120",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 101587,
        "difficulty": null
      },
      {
        "id": "3fa9eb77631daed1eb23102655b35d6e972dbf05864a5307325816a83f660aec",
        "domain": "numina",
        "question": "10. (20 points) Given the function\n$$\nf(x)=x^{4}+a x^{3}+b x^{2}+a x+1(a, b \\in \\mathbf{R})\n$$\n\nhas at least one root. Find the minimum value of $a^{2}-b$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142582,
        "difficulty": null
      },
      {
        "id": "7a4f4fbb77d0e81a9d82a60c01b773fa722915f3c6f09e8adb64996c884eaf2d",
        "domain": "numina",
        "question": "【Question 10】\nAs shown in the figure, in square $A B C D$, extend $B A$ to $G$ such that $A G=B D$. Then the degree measure of $\\angle B C G$ is $\\qquad$ degrees.",
        "answer": "67.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 135522,
        "difficulty": null
      },
      {
        "id": "4f34a3265b5254d6e39f75696670db2bfddafffff26f383884b0a8f837067c9c",
        "domain": "numina",
        "question": "4. The function $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ is given such that $f(n)=1$ if $n$ is odd, and $f(n)=k$ for every even number $n=2^{k} l$, where $k$ is a natural number and $l$ is an odd number. Determine the largest natural number $n$ for which\n\n$$\nf(1)+f(2)+\\ldots+f(n) \\leqq 123456 .\n$$",
        "answer": "82307",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75645,
        "difficulty": null
      },
      {
        "id": "62d1ad3a7d2bd66fa51ddd060effc5c117a0daa9b6d60e71655d9e58297df394",
        "domain": "numina",
        "question": "27. Find the digit that has the property that if it is appended to the end of any natural number, the resulting number is equal to the sum of three terms, the first of which is the original number, the second is the number represented by the sought digit, and the third is the product of the first two terms.",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39296,
        "difficulty": null
      },
      {
        "id": "e0f7b5b9f67e239c7139c259008f1299004971cc4222b90c409a56fe69808549",
        "domain": "numina",
        "question": "Misha thought of a number not less than 1 and not more than 1000. Vasya is allowed to ask only such questions, to which Misha can answer \"yes\" or \"no\" (Misha always tells the truth). Can Vasya determine the guessed number in 10 questions?",
        "answer": "183",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 72312,
        "difficulty": null
      },
      {
        "id": "37dc728a2185234ebbc7a18d4b063af609384a1bd5174d9f5303a9ce92ebdba7",
        "domain": "numina",
        "question": "11. Given that when $x \\in[0, \\sqrt{3}]$, $\\left|x^{3}+\\frac{3}{2}(1-a) x^{2}-3 a x+b\\right| \\leq 1$ always holds,\n(1) Prove: The equation $x^{3}+\\frac{3}{2}(1-a) x^{2}-3 a x+b=0$ has three real roots $x_{1}<x_{2}<x_{3}$,\n(2) Find $\\left(x_{1}^{2}-2-x_{2}\\right)\\left(x_{2}^{2}-2-x_{3}\\right)\\left(x_{3}^{2}-2-x_{1}\\right)$.",
        "answer": "-9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130487,
        "difficulty": null
      },
      {
        "id": "c90da1d85e0e23f46bbfa9d85e9850471d31e7a472626a0ef6304db79c7ca6a7",
        "domain": "numina",
        "question": "4. For a positive integer $n$, we denote by $P(n)$ the product of the positive divisors of $n$. For example, $P(20)=8000$. The positive divisors of 20 are 1, 2, 4, 5, 10, and 20, with the product $1 \\cdot 2 \\cdot 4 \\cdot 5 \\cdot 10 \\cdot 20=8000$.\n\n(a) Find all positive integers $n$ for which $P(n)=15 n$.\n\n(b) Show that there are no positive integers $n$ for which $P(n)=15 n^{2}$.",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85766,
        "difficulty": null
      },
      {
        "id": "74d63947c3904d3164c453fad1f3a672b23bb8586a2d8793e7a865b7902b9bfd",
        "domain": "numina",
        "question": "1. Given a set of data consisting of seven positive integers, the only mode is 6, and the median is 4. Then the minimum value of the sum of these seven positive integers is $\\qquad$ .",
        "answer": "26",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 95103,
        "difficulty": null
      },
      {
        "id": "70b60e1eb8b97c787d3e676520d6a434662ec574de105bd0155a56612f0a4e98",
        "domain": "numina",
        "question": "2. (5 points) On Valentine's Day, every student in the school gave each female student a valentine. It turned out that the number of valentines was 28 more than the total number of students. How many valentines were given?",
        "answer": "60",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19409,
        "difficulty": null
      },
      {
        "id": "39fb9d3862b776588031ca00f4a85ba41e4ad2daa204ecfd84f912393f558663",
        "domain": "numina",
        "question": "16 (12 points) Given in the cube $A B C D$ $A_{1} B_{1} C_{1} D_{1}$, $O, E, F, G$ are the midpoints of $B D$, $B B_{1}$, $A_{1} D_{1}$, $D_{1} C_{1}$ respectively, and $A B=1$. Find the volume of the tetrahedron $O E F G$.",
        "answer": "\\frac{5}{48}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127068,
        "difficulty": null
      },
      {
        "id": "93a498e21df9c757a2512d490ba195c6c4318c281a3d5d01c919b7cae1793cc8",
        "domain": "numina",
        "question": "2. (7 points) In Sun City, they exchange a honey cake for 6 pretzels, and for 9 pretzels, they give 4 doughnuts. How many doughnuts will they give for 3 honey cakes? Explain your answer.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13202,
        "difficulty": null
      },
      {
        "id": "22bb9dc01ed756866646ac5df64f743d29ce3c9cd0ac664360c1abbd89810fd0",
        "domain": "numina",
        "question": "Example 1. Derive the Taylor formula for $n=2$ for the function $f(x, y)=x^{y}$ in the neighborhood of the point $M_{0}(1,1)$ and approximate $1.1^{1.02}$.",
        "answer": "1.102",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30279,
        "difficulty": null
      },
      {
        "id": "c2ac5669ecb1908dab5e3764253e86bcb773d4472a76dc9d76791fe705fc701b",
        "domain": "numina",
        "question": "Prokopenko D:\n\nAround triangle $A B C$, a circle $\\Omega$ is described. Let $L$ and $W$ be the points of intersection of the angle bisector of angle $A$ with side $B C$ and circle $\\Omega$, respectively. Point $O$ is the center of the circumcircle of triangle $A C L$. Restore triangle $A B C$ if the circle $\\Omega$ and points $W$ and $O$ are given.",
        "answer": "0",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58515,
        "difficulty": null
      },
      {
        "id": "8ae4141ae245bb375b85f0894e72b84f2fbd7ed0c0910a2c3e7c9ad43ead0e48",
        "domain": "numina",
        "question": "\\section*{Problem 6B - 111236B}\n\n50 white and 50 black balls are to be distributed into two indistinguishable urns in such a way that neither urn remains empty and all balls are used.\n\nHow should the balls be distributed between the two urns so that the probability of drawing a white ball when one of the two urns is chosen at random and a ball is drawn from it is as high as possible?\n\n\\section*{Hints:}\n\n1. In classical probability theory, the probability \\(p\\) of an event is defined as the quotient of the number \\(g\\) of favorable cases for this event and the total number \\(m\\) of possible cases, i.e., \\(p=\\frac{g}{m}\\).\n\n2. Therefore, the probability of drawing a white ball (blindly) from an urn that contains a total of \\(u\\) balls, of which \\(w\\) are white, is set to \\(p=\\frac{w}{u}\\).\n\n3. If there are two urns, with the probability of drawing a white ball being \\(p_{1}\\) and \\(p_{2}\\) respectively, then the probability of the combined event \"selecting one of the two urns and drawing a white ball from the selected urn\" is \\(p=\\frac{1}{2} p_{1}+\\frac{1}{2} p_{2}\\).",
        "answer": "\\frac{74}{99}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 185814,
        "difficulty": null
      },
      {
        "id": "98c484e248f21c2e04f0c88c9012f0188f63e1dec7a235781aab28ff911311f7",
        "domain": "numina",
        "question": "\\section*{Problem 5 - 071035}\n\nFor which real \\(a\\) does the sum of the squares of the solutions of the equation \\(x^{2}+a x+a-2=0\\) attain its smallest value?",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184977,
        "difficulty": null
      },
      {
        "id": "3bedec0e338d974b6a4dce104f9722b0faa29ad1dab1ecb6bfb0ca7879686642",
        "domain": "numina",
        "question": "2. In a small fishing game, each time you catch a small fish or a small turtle. If you catch a small fish, you get 5 points, and if you catch a small turtle, you lose 2 points. Luo Xiahei played 17 times, and his score increased from 100 points to 157 points. Out of these 17 times, he caught a small fish $\\qquad$ times.",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109293,
        "difficulty": null
      },
      {
        "id": "8717eaa6f224bd0e49bfc57ab27188b8ded21ace796bada8d37f0cbe25a4803e",
        "domain": "numina",
        "question": "2. $[\\mathbf{3}] A, B, C$, and $D$ are points on a circle, and segments $\\overline{A C}$ and $\\overline{B D}$ intersect at $P$, such that $A P=8$, $P C=1$, and $B D=6$. Find $B P$, given that $B P<D P$.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150866,
        "difficulty": null
      },
      {
        "id": "afe366983e87a0b70dc209949c23ece120e1f687768ededb0ed7a731d551e5a7",
        "domain": "numina",
        "question": "42. Someone asked Uncle Wang: \"How many pots of flowers are there in your garden?\" Uncle Wang said: \"Half are roses, a quarter are lilies, one seventh are peonies, and less than 6 pots are roses.\" Therefore, Uncle Wang's garden has $\\qquad$ pots of flowers.",
        "answer": "28",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 101562,
        "difficulty": null
      },
      {
        "id": "5da5c186608b7fbfeecb37cc1a56b60acdfd80ed98add30beee881659a7df5d9",
        "domain": "numina",
        "question": "4. In triangle $A B C$, points $M$ and $N$ are the midpoints of sides $A C$ and $B C$ respectively. It is known that the point of intersection of the medians of triangle $A M N$ is the point of intersection of the altitudes of triangle $A B C$. Find the angle $A B C$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15565,
        "difficulty": null
      },
      {
        "id": "0d467b49773ce5bdee223a57a86d6ae15d6ebd40dce59161aa43f05b08d776b9",
        "domain": "numina",
        "question": "Example 2.7 Among the integers from 1 to 1000, how many integers are divisible by 4 but not by 3 or 10?",
        "answer": "133",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 108571,
        "difficulty": null
      },
      {
        "id": "4e8cadd2c346908c4e37143c2cf7d497bf4c392a9c497b1f75547ccada9614ed",
        "domain": "numina",
        "question": "1. (16 points) Solve the equation $x-5=\\frac{3 \\cdot|x-2|}{x-2}$. If the equation has multiple roots, write their sum in the answer.\n\n#",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4250,
        "difficulty": null
      },
      {
        "id": "824dd39e466a70243fb075cef90fc7ced2fcd938013a629711fe05d22609d55a",
        "domain": "numina",
        "question": "Arnaldo, Bernaldo, Cernaldo, Dernaldo, and Ernaldo are students from different parts of Brazil who were chosen to represent their country in international olympiads. After several weeks of training, some friendships were formed. We then asked each of them how many friends they had made in the group. Arnaldo, Bernaldo, Cernaldo, and Dernaldo answered, respectively, that they had made 1, 2, 3, and 4 friends within the group. How many of the group members are friends with Ernaldo?",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 73873,
        "difficulty": null
      },
      {
        "id": "20c036b9ec660a09b02f1a93fd80374b257dee0d6838e844f35ea633be8cfb52",
        "domain": "numina",
        "question": "5. In the quadrilateral pyramid $P-ABCD$, $BC \\parallel AD$, $AD \\perp AB$, $AB=2\\sqrt{3}$, $AD=6$, $BC=4$, $PA=PB=PD=4\\sqrt{3}$. Then the surface area of the circumscribed sphere of the tetrahedron $P-BCD$ is $\\qquad$ .",
        "answer": "80\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143972,
        "difficulty": null
      },
      {
        "id": "ae037ed4063c5d7afe52c466301e40a9711121f28701c26c117a26225a2765fe",
        "domain": "numina",
        "question": "# Problem 7. (4 points)\n\nTwo cubes with an edge of $12 \\sqrt[4]{\\frac{8}{11}}$ share a common face. A section of one of these cubes by a certain plane is a triangle with an area of 16. The section of the other by the same plane is a quadrilateral. What is the maximum value that its area can take?",
        "answer": "128",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20826,
        "difficulty": null
      },
      {
        "id": "38ac72633407d5d58fcc480819c0d3a002ea8e6b4aa0ed6c925465002339a302",
        "domain": "numina",
        "question": "$82009 \\times 1911$ chessboard's diagonal crosses $\\qquad$ squares.\n\nThe diagonal of an $82009 \\times 1911$ square grid crosses $\\qquad$ grid cells.",
        "answer": "83871",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 123583,
        "difficulty": null
      },
      {
        "id": "8a094a1179bfecdecbc3ae6566ff833876ff37cc7549ca2133d8b40bf495aa25",
        "domain": "numina",
        "question": "4. Let $p(x)$ be a polynomial of degree $2n$, $p(0)=p(2)=\\cdots=p(2n)=0, p(1)=p(3)=\\cdots=$ $p(2n-1)=2, p(2n+1)=-30$. Find $n$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102246,
        "difficulty": null
      },
      {
        "id": "372056739be60f9ba39a5987756458afffefa1ede66bea61f63f58930c0b2c5c",
        "domain": "numina",
        "question": "## Task 1 - 100931\n\nGünter spent a number of days in a camp with his FDJ group during his vacation. Every day, exactly two students from his group were assigned to table duty in the morning and exactly two students in the afternoon. Over the days, all students in his group were assigned to this table duty equally often.\n\nFurthermore, the following is known:\n\n(1) Günter was assigned to table duty on exactly 6 days.\n\n(2) If he had table duty in the afternoon, he did not have it in the morning.\n\n(3) He did not have table duty in the afternoon on exactly 13 of these days.\n\n(4) He did not have table duty in the morning on exactly 11 of these days.\n\nHow many students were in Günter's group?",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 184488,
        "difficulty": null
      },
      {
        "id": "10630166f8f94c92a43bddc5cf78ecfc1ff68f5acaebdb21eaa893a7e583eeff",
        "domain": "numina",
        "question": "## Task 3 - 010913\n\nTo maintain the cutting ability of tools when machining metals, cooling is often done with an emulsion of lubricated mineral oil (density $0.98 \\mathrm{~g} / \\mathrm{cm}^{3}$) and as soft water as possible (density $1.0 \\mathrm{~g} / \\mathrm{cm}^{3}$). The mixture must have a density of $0.996 \\mathrm{g} / \\mathrm{cm}^{3}$ for cutting tools of higher hardness, and a density of $0.992 \\mathrm{~g} / \\mathrm{cm}^{3}$ for grinding operations. How many liters of lubricated mineral oil and how many liters of soft water are needed for 10 liters of emulsion in each case?",
        "answer": "81",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184363,
        "difficulty": null
      },
      {
        "id": "1d0db737bee1e577d0ff4543fe94846aa540e669a73f13ce0678ebd7e05267bd",
        "domain": "numina",
        "question": "In the drawing below, $\\triangle A B C$ is an equilateral triangle and $C D$ is both a height of the triangle and a diameter of the circle. If $A B=10 \\mathrm{~cm}$, determine the shaded area.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-02.jpg?height=426&width=491&top_left_y=495&top_left_x=680)",
        "answer": "\\frac{100\\pi-75\\sqrt{3}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73515,
        "difficulty": null
      },
      {
        "id": "97878c822e8739cd37fb601c4d74a57e2eda122ea11e7f9de581b9464917cfa1",
        "domain": "numina",
        "question": "5. (1 mark) A positive integer is said to be a \"palindrome\" if it reads the same from left to right as from right to left. For example 2002 is a palindrome. Find the sum of all 4-digit palindromes.\n(1 分) A positive integer is said to be a \"palindrome\" if it reads the same from left to right as from right to left. For example 2002 is a palindrome. Find the sum of all 4-digit palindromes.",
        "answer": "495000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153450,
        "difficulty": null
      },
      {
        "id": "032ea580351c661e91c54baaae7db84f5b9006ee37ce9c2803638f87a6ebf96c",
        "domain": "numina",
        "question": "5. Let $E$ be a moving point inside square $ABCD$. It is known that the minimum value of the sum of the distances from $E$ to points $A$, $B$, and $C$ is $\\sqrt{2}+\\sqrt{6}$. Try to find the side length of this square.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105296,
        "difficulty": null
      },
      {
        "id": "0555b372147b8dea84f18a035fc82e4452817f01ba8dd3c2df09787deb054039",
        "domain": "numina",
        "question": "The rhombus $ABCD$ is divided into two equilateral triangles by its diagonal $BD$. A point $P$ is given on the segment $AD$, and a point $Q$ is on the segment $CD$ such that $\\angle PBQ = 60^{\\circ}$. What are the other two angles of the triangle $PBQ$?",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171392,
        "difficulty": null
      },
      {
        "id": "42ffa32c2230dc02aaaffc6b3296c77927dd4f9fb669a342d48a0268682c1f5f",
        "domain": "numina",
        "question": "16. (15 points) Given the functions\n$$\nf(x)=a \\sin x(a \\in \\mathbf{R}), g(x)=\\mathrm{e}^{x} \\text {. }\n$$\n(1) If $00, m0$ always holds, find the minimum value of $k$.",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 102990,
        "difficulty": null
      },
      {
        "id": "1c06fc0f4f8b0d63fd1403c4582a09580fb0fc505dfa14e7bc58c6d1149d7cd9",
        "domain": "numina",
        "question": "Problem 5.1. After a football match, the coach lined up the team as shown in the figure, and commanded: \"Run to the locker room, those whose number is less than that of any of their neighbors.\" After several people ran away, he repeated his command. The coach continued until only one player was left. What is Igor's number, if it is known that after he ran away, 3 people remained in the line? (After each command, one or several players ran away, after which the line closed, and there were no empty spaces between the remaining players.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_adefdf2067937cb67fc7g-08.jpg?height=386&width=1072&top_left_y=485&top_left_x=191)",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18153,
        "difficulty": null
      },
      {
        "id": "ff04bf8b96795b93f7616a2eb2302676f4d9e135fc30c3b65242387774fbe5d4",
        "domain": "numina",
        "question": "\\section*{Problem 1 - 181041}\n\nWhat are the last two digits (in the usual decimal digit notation) of the number \\(x\\) that satisfies the equation\n\n\\[\n\\log _{13}\\left[\\log _{12}\\left(\\log _{11} x\\right)\\right]=1\n\\]",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 185188,
        "difficulty": null
      },
      {
        "id": "990d27f2860bca972487dc55067d6d88638bc9bd45bd53a59dbfe6aca4332f5b",
        "domain": "numina",
        "question": "Two parallel lines are intersected by a third. Find the angle between the bisectors of the internal same-side angles.\n\n#",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62340,
        "difficulty": null
      },
      {
        "id": "d4d14312eec8f536053a947cd9d021bbec930a75123ec94088fb83dea50ddf06",
        "domain": "numina",
        "question": "31. The blackboard has the following 9 numbers written on it. Each time, the teacher arbitrarily erases two numbers, and writes their sum on the blackboard. When only one number remains on the blackboard, this number is $\\qquad$.",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127815,
        "difficulty": null
      },
      {
        "id": "7e02cf6c4fe1eefa6ea5e7a54291020e76b1f6ae15351c40fd6a84c4243bcc0f",
        "domain": "numina",
        "question": "9.6. In a certain company, there are 100 shareholders, and any 66 of them own no less than $50 \\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own?",
        "answer": "25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5716,
        "difficulty": null
      },
      {
        "id": "4f38384bd94151b0e9bce5e7c048fa6c6ef48d746f89cecd91c412353a9ace8c",
        "domain": "numina",
        "question": "In triangle $ABC$, construct points $C_1$ on side $AB$ and $A_1$ on side $BC$ such that the lines $AA_1$ and $CC_1$ intersect at point $P$ where $AP / PA_1 = 3 / 2$ and $CP / PC_1 = 2 / 1$. In what ratio does point $C_1$ divide side $AB$?",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 164309,
        "difficulty": null
      },
      {
        "id": "77656dff19cbe640780da813fcbc5955c420a12dd023d24debd0b95d9c5b8883",
        "domain": "numina",
        "question": "10. Let $A B C D$ be a square of side length 13 . Let $E$ and $F$ be points on rays $A B$ and $A D$, respectively, so that the area of square $A B C D$ equals the area of triangle $A E F$. If $E F$ intersects $B C$ at $X$ and $B X=6$, determine $D F$.",
        "answer": "\\sqrt{13}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151300,
        "difficulty": null
      },
      {
        "id": "0e839c02749f7e80a94c01485f9c21ff06f7b364a7c077c2c700a4be25f7de7f",
        "domain": "numina",
        "question": "2. If the difference between the maximum and minimum elements of the real number set $\\{1,2,3, x\\}$ equals the sum of all elements in the set, then the value of $x$ is $\\qquad$ .",
        "answer": "-\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92326,
        "difficulty": null
      },
      {
        "id": "d32f07ac2d323c531c94da75b8b10adcb0c04af31040857f223ac90511633506",
        "domain": "numina",
        "question": "2. Each card has a number on one side and a letter on the other side (two cards that have the same letter on one side do not necessarily have the same number on the other side, and vice versa). On the table, there are cards that show the following on their visible side:\n\n$$\nM, A, T, E, M, A, T, И, K, A, 2,0,1,8\n$$\n\nVlada claims the following: \"If a card has a vowel, then the other side has an even number.\" Miljan wants to check Vlada's claim. How many and which cards does he need to turn over to verify the correctness of the statement?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 161113,
        "difficulty": null
      },
      {
        "id": "eb3ecb18f646cb5f980a21a656429decf1186b7d68392e4e80fe63250a09806b",
        "domain": "numina",
        "question": "Tokarev S.i.\n\nIn a line, all integers from 1 to 100 are written in an unknown order. With one question about any 50 numbers, you can find out the order of these 50 numbers relative to each other. What is the minimum number of questions needed to definitely determine the order of all 100 numbers?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56728,
        "difficulty": null
      },
      {
        "id": "ca024ac51114a748b72dcb150a933cfe3d93db3c30fe9495a1820d9ed47bd0da",
        "domain": "numina",
        "question": "$A$ says to $B$: »I will tell you how old you are if you answer the following question. Multiply the digit in the tens place of your birth year by 5, add 2 to this product, and then multiply the resulting sum by 2. Add the units digit of your birth year to this new product and tell me the number you get!«\n\n$B$ replies: »43«, whereupon $A$ immediately exclaims: »You are 16 years old!« How did $A$ know this?",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 175686,
        "difficulty": null
      },
      {
        "id": "57466e70d611206e2cd45de3006f14a24c0501fe1820a1d66073ce6eae3b18fb",
        "domain": "numina",
        "question": "1. A tractor is pulling a very long pipe on sled runners. Gavrila walked along the entire pipe at a constant speed in the direction of the tractor's movement and counted 210 steps. When he walked in the opposite direction at the same speed, the number of steps was 100. What is the length of the pipe if Gavrila's step is 80 cm? Round the answer to the nearest whole number of meters. The speed of the tractor is constant.",
        "answer": "108",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11394,
        "difficulty": null
      },
      {
        "id": "e4254956063e2b8607f4712ba86c711b0cf4f9a6ae072d6d95e825b487a90b26",
        "domain": "numina",
        "question": "The National Bank of Hungary withdrew the 1 and 2 forint coins from circulation on March 1, 2008. In cash purchases in stores, the prices of individual items are not rounded separately, but the final amount is rounded to the nearest five. The following table can be seen in the issued information about the rounding rules:\n\namounts ending in 1 or 2 are rounded down to 0;\n\namounts ending in 3 or 4 are rounded up to 5;\n\namounts ending in 6 or 7 are rounded down to 5;\n\namounts ending in 8 or 9 are rounded up to 0.\n\nMarci buys two croissants every morning at the local bakery. After a few days, due to rounding, he saves exactly the price of one croissant. We know that the price of a croissant is more than 10 Ft. What could be the unit price of the croissant if this saving occurs as quickly as possible?",
        "answer": "16",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 170365,
        "difficulty": null
      },
      {
        "id": "34e84733149116946e44ea3f48f213d0f1a9798fc0e2f371b3d3b1b5ee0ea3f4",
        "domain": "numina",
        "question": "2. Given positive integers $a, b, c (a \\leqslant b \\leqslant c)$ and real numbers $x, y, z, w$ satisfying $a^{x}=b^{y}=c^{z}=70^{w}$, and $\\frac{1}{x}+\\frac{1}{y}+\\frac{1}{z}=\\frac{1}{w}$, the value of $c$ is $\\qquad$.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126042,
        "difficulty": null
      },
      {
        "id": "e4af15d4006383f7d42fbc1872930ec173e22be6ba981c66734a18e940243aa1",
        "domain": "numina",
        "question": "The first and second terms of a sequence are 4 and 5 , respectively. Each term after the second is determined by increasing the previous term by one and dividing the result by the term before that. For example, the third term equals $\\frac{5+1}{4}$. What is the 1234 th term of this sequence?",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194415,
        "difficulty": null
      },
      {
        "id": "2b257ab7dfa4093ece6405fca16ae34ceab8000621709acb8d31adcfda54fb2d",
        "domain": "numina",
        "question": "5. The sum of the approximate values you get when you round the largest four-digit number with distinct odd digits to the nearest hundred, and the smallest four-digit number with distinct even digits to the nearest ten, and then round to the nearest thousand.",
        "answer": "11000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87006,
        "difficulty": null
      },
      {
        "id": "03101746fb197ea0aa57242c0d1688b73b1930fc7bbf52f047b6079fbca8cd8c",
        "domain": "numina",
        "question": "If we want to write down all the integers from 1 to 10,000, how many times do we need to write a digit, for example, the digit 5?",
        "answer": "4000",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 176469,
        "difficulty": null
      },
      {
        "id": "7792bd199b945e8af5344550441f9e6af53d7479255868a2b2a089d2c90f4f19",
        "domain": "numina",
        "question": "2. (8 points) A shepherd was herding a flock of sheep to graze. After one ram ran out, he counted the number of sheep and found that the ratio of rams to ewes among the remaining sheep was 7:5. After a while, the ram that had run out returned to the flock, but then a ewe ran out. The shepherd counted the sheep again and found that the ratio of rams to ewes was 5:3. How many sheep were there in the flock originally?",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111973,
        "difficulty": null
      },
      {
        "id": "b5bdfb96feb17ddff5b62d3adb9156e2a065ffd22f74e76880e340e9f4d5f1b8",
        "domain": "numina",
        "question": "2. (5 points) Two different natural numbers end with 7 zeros and have exactly 72 divisors. Find their sum.\n\n#",
        "answer": "70000000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19551,
        "difficulty": null
      },
      {
        "id": "35e68939c5ff4531abcea1e69204b56c589ae8fa519ee3138280d9eea6e757d0",
        "domain": "numina",
        "question": "111. In the pyramid $S A B C$, the products of the lengths of the edges of each of the four faces are equal to the same number. The length of the height of the pyramid, dropped from $S$ to the face $A B C$, is $2 \\sqrt{\\frac{102}{55}}$, and the measure of the angle $C A B$ is $\\arccos \\left(\\frac{1}{6} \\sqrt{\\frac{17}{2}}\\right)$. Find the volume of the pyramid $S A B C$, if\n\n$$\n|S A|^{2}+|S B|^{2}-5|S C|^{2}=60\n$$",
        "answer": "\\frac{34\\sqrt{6}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29820,
        "difficulty": null
      },
      {
        "id": "0d09f7b6eaa6ebbf57c969db58fd77c7a52981c43cc0f8741b2fc2717fbeda11",
        "domain": "numina",
        "question": "5. Through the vertex $M$ of some angle, a circle is drawn, intersecting the sides of the angle at points $N$ and $K$, and the bisector of this angle at point $L$. Find the sum of the lengths of segments $M N$ and $M K$, if the area of $M N L K$ is 25, and the angle $L M N$ is $30^{\\circ}$.",
        "answer": "10\\sqrt[4]{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8543,
        "difficulty": null
      },
      {
        "id": "bf0f57b833797ba0983a22a20155e4be3dc718d7f6009d8d9950ff461144fa1b",
        "domain": "numina",
        "question": "28. Let's determine the amount of sulfur substance\n\nAccording to the chemical reaction equation:\n\n$$\n\\mathrm{n}(\\mathrm{S})=\\mathrm{n}\\left(\\mathrm{SO}_{2}\\right)=1 \\text { mole }\n$$\n\n29. Let's calculate the mass of pure sulfur:\n\n$$\nm(S)=n \\cdot M=1 \\cdot 32=32 \\text { g }\n$$\n\nDetermination of the mass of the substance - 2 points\n\n30. Thus, in a sample of natural sulfur weighing 38 g, there are 32 g of pure substance.\n\nLet's calculate the mass of the impurity:\n\n$m($ impurity $)=38-32=6$ g\n\nDetermination of the mass of the impurity - 2 points",
        "answer": "6",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 7246,
        "difficulty": null
      },
      {
        "id": "bf67fb1bc43b52945944609bcdbba0833ecf87315e1048dd669addb8db777fe4",
        "domain": "numina",
        "question": "## Task A-1.1.\n\nAna and Vanja are standing together next to a railway track and waiting for a train that is moving at a constant speed. At the moment when the front of the train reaches them, Ana starts walking at a constant speed in the direction of the train, while Vanja walks at the same speed in the opposite direction. Each of them stops at the moment when the rear of the train passes by her. Ana has walked a total of 45 meters, while Vanja has walked 30 meters. How long is the train?",
        "answer": "180",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87279,
        "difficulty": null
      },
      {
        "id": "145f94eebaf4e800d04ba10509dc832000ad5c7514ac7003f8f69efd469ddef4",
        "domain": "numina",
        "question": "16. (6 points) If the two-digit number $\\overline{a b}$ is 24 more than $\\overline{c d}$, and the three-digit number $\\overline{1 a b}$ is 15 more than $\\overline{c d 1}$, then $\\overline{a d}=$",
        "answer": "32",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 133224,
        "difficulty": null
      },
      {
        "id": "42fb1293f6fdf295919424eba047d05e4d3917c660bed8172453774e43dda4ae",
        "domain": "numina",
        "question": "8. (10 points) In the expression $(x+y+z)^{2018}+(x-y-z)^{2018}$, the brackets were expanded and like terms were combined. How many monomials $x^{a} y^{b} z^{c}$ with a non-zero coefficient were obtained?",
        "answer": "1020100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19503,
        "difficulty": null
      },
      {
        "id": "9ce5438f28ffd36e90643daec996a814aea67bccfd5a6450cf13625fcdae8b3e",
        "domain": "numina",
        "question": "1. From Central Square to the station, there is a straight street divided by 11 intersections into 12 equal blocks. At each intersection, there is a traffic light. All traffic lights simultaneously turn green for 3 minutes, then red for 1 minute. It takes the bus two minutes to travel one block (from intersection to intersection), while a car takes one minute to travel one block. The bus and the car start from the square at the same time, at which moment all traffic lights turn green. Which vehicle will arrive at the station first, and how many minutes earlier?",
        "answer": "9",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16563,
        "difficulty": null
      },
      {
        "id": "c3a79c7acc7e2c6c190bf5a36d7e123412e0024e3136e64b8c301c79c4c3dc8e",
        "domain": "numina",
        "question": "[  Angles between angle bisectors  Law of Cosines\n\n[ Area of a triangle (using semiperimeter and radius of inscribed or excircle\n\nIn triangle $A B C$, the angle at vertex $B$ is $\\frac{\\pi}{3}$, and the segments connecting the incenter to vertices $A$ and $C$ are 4 and 6, respectively. Find the radius of the circle inscribed in triangle $A B C$.\n\n#",
        "answer": "\\frac{6\\sqrt{3}}{\\sqrt{19}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62763,
        "difficulty": null
      },
      {
        "id": "8dacd2c625cebb6619a42f9a545a77b634f60ef04c56a5fc0ea8bc2e012ec6f5",
        "domain": "numina",
        "question": "8. The smallest solution to the equation $\\left[\\frac{x}{2}\\right]+\\left[\\frac{x}{3}\\right]+\\left[\\frac{x}{7}\\right]=x$ is $\\qquad$ (where, $[x]$ denotes the greatest integer not exceeding $x$).",
        "answer": "-85",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144201,
        "difficulty": null
      },
      {
        "id": "b2552c9810e1493ac5c636cbfdb1c02ffdf2ea2650523782337f925eb352e322",
        "domain": "numina",
        "question": "53. What is the smallest number of non-overlapping tetrahedra into which a cube can be divided?",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 25262,
        "difficulty": null
      },
      {
        "id": "7f32b63ce9f9813cf076fb50eb83ce6ffd406a1fc2c5797627d61a88c40b82d2",
        "domain": "numina",
        "question": "2. Let $a_{i}(i=1,2,3,4)$ be real numbers. If the sum of the elements of all non-empty proper subsets of the set $\\left\\{a_{1}, a_{2}, a_{3}, a_{4}\\right\\}$ is\n28, then $a_{1}+a_{2}+a_{3}+a_{4}=$ $\\qquad$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107186,
        "difficulty": null
      },
      {
        "id": "e128210fa954e36e30cfdc6a9d8d24d714de292a1e643c2df847ba427527801f",
        "domain": "numina",
        "question": "In acute $\\triangle ABC$, $AB > AC$, $\\angle ACB - \\angle ABC = 30^{\\circ}$, $O$ and $H$ are the circumcenter and orthocenter of $\\triangle ABC$ respectively, and $\\triangle DOH$ is an equilateral triangle. Find the value of $\\angle DAO + \\angle AOH$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 123311,
        "difficulty": null
      },
      {
        "id": "2c8dc71a58e8d91f511af229c4921f8efb0fbb671a24db69b9cc01bbe97722b5",
        "domain": "numina",
        "question": "4. Let the tetrahedron $P-ABC$ satisfy $PA=PB=3, AB=BC=CA=2$, then the maximum volume of the tetrahedron is $\\qquad$ .",
        "answer": "\\frac{2\\sqrt{6}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102860,
        "difficulty": null
      },
      {
        "id": "ccabd0d82734548e5786a6c55ea43ebac51519b756ff62d3dd8694797a2038e2",
        "domain": "numina",
        "question": "## Task 1 - 050811\n\nA student had attached the following overview of participation in the 1st stage of the 4th Olympiad of Young Mathematicians of the GDR to the wall newspaper:\n\nClass 8a: Out of 33 students, 20 participated, which is approximately 60.6 percent.\n\nClass 8b: Out of 32 students, 21 participated, which is approximately 65.6 percent.\n\nClass 8c: Out of 27 students, 19 participated, which is approximately 70.4 percent.\n\nThe students of these classes are tasked with determining the overall percentage of participation of the 8th grade students. Some students calculate the arithmetic mean of the percentages, while others calculate the quotient of the total number of participants and the total number of students in these classes, multiplied by 100.\n\na) How large is the difference that results from the two calculations?\n\nb) Which students have calculated the percentage in the correct way?",
        "answer": "0.31",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183969,
        "difficulty": null
      },
      {
        "id": "7f3395f4d8c8e04067eee2217878f4324290593233ad3abbfed9b72b85d5cbb8",
        "domain": "numina",
        "question": "The function $f$ has the following properties:\n\n(i) its domain is all real numbers,\n\n(ii) it is an odd function (that is, $f(-x)=-f(x)$ for every real number $x$ ), and\n\n(iii) $f(2 x-3)-2 f(3 x-10)+f(x-3)=28-6 x$ for every real number $x$.\n\nDetermine the value of $f(4)$.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194060,
        "difficulty": null
      },
      {
        "id": "85517f315a90803b19b552d094e196c4da5f601f51f5d7162c82a290145ae6b6",
        "domain": "numina",
        "question": "【Question 2】\nThe sum of two numbers is 17, the difference between the two numbers is 7, the larger number is $\\qquad$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144818,
        "difficulty": null
      },
      {
        "id": "f90cb0dbfe6df4c7a4580d28553a00ab326a049d4fa4c2e7fd36956c145b3e87",
        "domain": "numina",
        "question": "16. Let $n$ be a given positive integer, $S_{n} \\subseteq\\left\\{\\alpha \\mid \\alpha=\\left(p_{1}, p_{2}, \\cdots, p_{n}\\right), p_{k} \\in\\{0,1\\}, k=1,2, \\cdots, n\\right\\}$. For any elements $\\beta=\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right)$ and $\\gamma=\\left(y_{1}, y_{2}, \\cdots, y_{n}\\right)$ in the set $S_{n}$, the following conditions are satisfied:\n(1) If $\\beta=\\gamma$, then $\\sum_{i=1}^{n} x_{i} y_{i}$ is odd;\n(2) If $\\beta \\neq \\gamma$, then $\\sum_{i=1}^{n} x_{i} y_{i}$ is even.\n(1) When $n=5$, find the maximum number of elements in the set $S_{5}$;\n(2) When $n \\geqslant 6$, find the maximum number of elements in the set $S_{n}$.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 114641,
        "difficulty": null
      },
      {
        "id": "d9d6e18eceebf60f818a3d30b430591e197dc55a6cc07ebe44b56045e0b83d23",
        "domain": "numina",
        "question": "10. In $\\triangle A B C, A B=9, B C=8$ and $A C=7$. The bisector of $\\angle A$ meets $B C$ at $D$. The circle passing through $A$ and touching $B C$ at $D$ cuts $A B$ and $A C$ at $M$ and $N$ respectively. Find $M N$.\n在 $\\triangle A B C$ 中, $A B=9, B C=8$ 及 $A C=7 \\circ \\angle A$ 的角平分線交 $B C$ 於 $D$ 。穿過 $A$且與 $B C$ 相切於 $D$ 的圓分別交 $A B$ 和 $A C$ 於 $M$ 和 $N \\circ$ 求 $M N$ 。",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153639,
        "difficulty": null
      },
      {
        "id": "4d2a58b85350de945d3819d859e4977192a481950ad9634adb9052838a3ea558",
        "domain": "numina",
        "question": "714. Find the limits:\n1) $\\lim _{\\substack{x \\rightarrow 3 \\\\ y \\rightarrow 0}} \\frac{\\tan(x y)}{y}$\n2) $\\lim _{\\substack{x \\rightarrow 0 \\\\ y \\rightarrow 0}} \\frac{x}{x+y}$.",
        "answer": "3",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 34295,
        "difficulty": null
      },
      {
        "id": "166f141b0f56c1cb1141fc0349f4f33f2a059e85da1253a985e03afbf831b90d",
        "domain": "numina",
        "question": "[Example 3.3.9] Let $n$ be a fixed integer, $n \\geqslant 2$\n(I) Determine the smallest constant $c$ such that the inequality\n$$\n\\sum_{1 \\leqslant i<j \\leqslant n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4} ;\n$$\n(II) For this constant $c$, determine the necessary and sufficient conditions for equality to hold.",
        "answer": "\\frac{1}{8}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 113594,
        "difficulty": null
      },
      {
        "id": "0b9b51da4c75b76342c9e2c35016b16c1f0ba4fc2e954ce17d3e262c38ea3cb7",
        "domain": "numina",
        "question": "## Subject 4.\n\nAna, Barbu, and Cristi together have an amount of money less than 100 lei, and each of them has more than 5 lei. Ana and Barbu together have three times more than Cristi, and Barbu and Cristi together have three times more than Ana. Find the largest and smallest amounts of money that the three children can have.\n\n## Note: All subjects are mandatory\n\nTime allowed: 2 hours\n\n## MATHEMATICS OLYMPIAD  LOCAL STAGE  JANUARY 18, 2014\n\n## GRADE 5  Scoring Guidelines",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178840,
        "difficulty": null
      },
      {
        "id": "9bb4e9d2a44a46461d42072ae76812288d864cf9dd8bcae79158c7e2ae4ed435",
        "domain": "numina",
        "question": "7.4. Karlson counts 200 buns baked by Fräulein Bock: «one, two, three, ..., one hundred and nine, one hundred and ten, two hundred». How many words will he say in total? (Each word is counted as many times as it was said.)",
        "answer": "443",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16818,
        "difficulty": null
      },
      {
        "id": "9afc27b0c491087a9a9103a4210f867570c2c6fa20d6d9e913793c290d081ebf",
        "domain": "numina",
        "question": "20. (6 points) If the ten-digit number $\\mathrm{a} 2016 \\mathrm{~b} 2017$ is divisible by 33, then, how many such ten-digit numbers are there?",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 132324,
        "difficulty": null
      },
      {
        "id": "acceccadcba94eae212615e55c84c89dc998febf195282a8cfa08e2d8d782ac8",
        "domain": "numina",
        "question": "B1. What is the maximum possible greatest common divisor of the numbers $a-2b+3, 2a-3b-1$, and $3a+b-2$, if $a$ and $b$ are natural numbers?",
        "answer": "38",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162999,
        "difficulty": null
      },
      {
        "id": "7d155af30b2cfa8a6716f0f38d28cc90b99a1af1da2e28bffd6a6ea4e02e1acd",
        "domain": "numina",
        "question": "6. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=5, A A_{1}=4, A D=3$, points $E, F, G$ are the midpoints of edges $A A_{1}, C_{1} D_{1}$, and $B C$, respectively. Then the volume of the tetrahedron $B_{1}-E F G$ is $\\qquad$",
        "answer": "\\frac{45}{16}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 111932,
        "difficulty": null
      },
      {
        "id": "b5eea57e9d6af53ec8e9a693a862edc7dda568a4273e4a29a188c1d1cabbbad3",
        "domain": "numina",
        "question": "Jindra collects dice, all of the same size. Yesterday he found a box and started arranging the dice in it. He managed to fill the square bottom with one layer of dice. He then filled five more layers in the same way, but ran out of dice halfway through the next layer. Today, Jindra received 18 more dice from his grandmother, which were exactly what he needed to complete this layer.\n\nHow many dice did Jindra have yesterday?\n\n(M. Petrová)\n\nHint. How many dice did Jindra have in each complete layer?",
        "answer": "234",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76892,
        "difficulty": null
      },
      {
        "id": "5f7d584f5b97fd589c968a177cb8eb69e87e57807de57df3ba4e8684e8a37059",
        "domain": "numina",
        "question": "6.2. In an isosceles triangle \\(ABC\\), one of the angles is equal to the difference of the other two, and one of the angles is twice another. The angle bisectors of angles \\(A\\), \\(B\\), and \\(C\\) intersect the circumcircle of the triangle at points \\(L\\), \\(O\\), and \\(M\\) respectively. Find the area of triangle \\(LOM\\), if the area of triangle \\(ABC\\) is 2. If the answer is not an integer, round it to the nearest integer.",
        "answer": "\\sqrt{3}+1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6467,
        "difficulty": null
      },
      {
        "id": "c9a6b0a9c37c3725d41e84d387c70f05233bcfdc31ef9bacbaf06788d28d6d20",
        "domain": "numina",
        "question": "1. For how many natural numbers $n$ are the inequalities\n\n$$\n2019<\\sqrt{n}<2020 ?\n$$\n\nExplain your answer!",
        "answer": "4038",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 189411,
        "difficulty": null
      },
      {
        "id": "9935cdcb9ed51c77140783697576472ca75e7b9e964b1ae85b5632c0c76d1c25",
        "domain": "numina",
        "question": "6. (10 points) Two cubes of different sizes are glued together to form the solid figure shown below, where the four vertices of the smaller cube's bottom face are exactly the midpoints of the edges of the larger cube's top face. If the edge length of the larger cube is 2, then the surface area of this solid figure is . $\\qquad$",
        "answer": "32",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102733,
        "difficulty": null
      },
      {
        "id": "f1f510313b7b7996be070e2875a29823a4d52eb3c3d4b94c8f887b3ec08fbffc",
        "domain": "numina",
        "question": "Bogognov i.i.\n\nIn some cells of a $200 \\times 200$ square, there is one chip - either red or blue; the other cells are empty. One chip sees another if they are in the same row or column. It is known that each chip sees exactly five chips of the other color (and possibly some chips of its own color). Find the maximum possible number of chips.",
        "answer": "3800",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55075,
        "difficulty": null
      },
      {
        "id": "7e33897131a170cc593e3e1f220b29d1a579d58f6180ba7bba8f4c48038aa869",
        "domain": "numina",
        "question": "14. A four-digit number, if a comma is placed between the hundreds and tens place, can be written as two two-digit numbers $(3126 \\rightarrow 31,62)$. If the two two-digit numbers have an integer multiple relationship, we call such a four-digit number a \"clever number\". If you select 4 numbers from $1, 2, 4, 6, 8$ to form a four-digit number, how many \"clever numbers\" are there? $\\qquad$",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 120772,
        "difficulty": null
      },
      {
        "id": "a2163d5847f871b8d5a286c7a5d141c9b0d0db4db0ce636c1e30ed3de8eedd8e",
        "domain": "numina",
        "question": "6. Let the hyperbola $\\frac{x^{2}}{a^{2}}-\\frac{y^{2}}{b^{2}}=1(a>0, b>0)$ have its left and right foci as $F_{1}$ and $F_{2}$, respectively, and let $A$ be a point on the asymptote of the hyperbola such that $A F_{2} \\perp F_{1} F_{2}$. The distance from the origin $O$ to the line $A F_{1}$ is $\\frac{1}{3}\\left|O F_{1}\\right|$. Then the eccentricity of the hyperbola is",
        "answer": "\\frac{\\sqrt{6}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 121092,
        "difficulty": null
      },
      {
        "id": "7049fb57682dcb4fdba6ade8cce9f7b384abb7913f76a3c3367143ec6ff338c4",
        "domain": "numina",
        "question": "8,9\n\nTwo tangents are drawn to a circle of radius 7 from a point located at a distance of 25 from the center.\n\nFind the distance between the points of tangency.",
        "answer": "13.44",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63262,
        "difficulty": null
      },
      {
        "id": "40c798dd3a771e9df148fcbf2b3cab740d038833c717bb230bd4d1211e1ca27d",
        "domain": "numina",
        "question": "16. Through the right focus $F$ of the ellipse $\\frac{x^{2}}{3}+\\frac{y^{2}}{2}=1$, draw two perpendicular chords $A B$ and $C D$. Let the midpoints of $A B$ and $C D$ be $M$ and $N$ respectively.\n(1) Prove that the line $M N$ must pass through a fixed point, and find this fixed point;\n(2) If the slopes of chords $A B$ and $C D$ both exist, find the maximum value of the area $S$ of $\\triangle F M N$.",
        "answer": "\\frac{4}{25}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 100799,
        "difficulty": null
      },
      {
        "id": "c43d2390cee1b83f056e48c47e45b3f0edf65a292d83532753ee65e4394ebbcc",
        "domain": "numina",
        "question": "2. Sergey wrote down the numbers from 500 to 1499 in a row in some order. Under each number, except the leftmost one, he wrote the GCD of that number and its left neighbor, obtaining a second row of 999 numbers. Then he applied the same rule to get a third row of 998 numbers, from it a fourth row of 997 numbers, and so on. He stopped when all the numbers in the next row were equal to one for the first time. What is the maximum number of rows he could have written by this moment?",
        "answer": "501",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16873,
        "difficulty": null
      },
      {
        "id": "390795bde9695b0c32efcc4f277948b35105d582b4c9ea585019f3c75134b67e",
        "domain": "numina",
        "question": "3. A cyclist is moving from place A to place B, and halfway between A and B is place C. The cyclist covered the distance from A to C in 4 hours, traveling at a speed of $18 \\mathrm{~km}$ per hour. How many hours will it take the cyclist to cover the distance from place C to B, if he travels at a speed of $12 \\mathrm{~km}$ per hour?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188672,
        "difficulty": null
      },
      {
        "id": "bff192b5ce1247bc670581da764059d66771f383a79842378b9a9a004f592a52",
        "domain": "numina",
        "question": "2. In the Cartesian coordinate system, the area of the region formed by points $(x, y)$ satisfying $|x|+|y|+|x-2| \\leqslant 4$ is\n$\\qquad$ (Supplied by An Zhenping)",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 101930,
        "difficulty": null
      },
      {
        "id": "d77ce842a83d0f675d1cc39e852b5635294d51596802023f11667dbfb603bb32",
        "domain": "numina",
        "question": "1. Given the integer sequence $\\left\\{a_{n}\\right\\}$ satisfying $a_{n}=a_{n-1}-a_{n-2}(n \\geqslant 3)$, if the sum of the first 1492 terms is 1985, and the sum of the first 1985 terms is 1492, find the sum of the first 2001 terms.",
        "answer": "986",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145818,
        "difficulty": null
      },
      {
        "id": "9ae61306b2d57c90e4b852fc4c73fc44dfb9a909ea3bed0f5ce37bdea56be9f5",
        "domain": "numina",
        "question": "A hemisphere with radius $r=1$ is to be divided into two equal volume parts by a plane parallel to the base. How high is each of the two parts?",
        "answer": "0.6527",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 174275,
        "difficulty": null
      },
      {
        "id": "857b313539472bd292758a8fed94de13bb0eb066a54780b3e62455f272e26a8b",
        "domain": "numina",
        "question": "8. A station has exactly one bus arriving every $8: 00 \\sim 9: 00, 9: 00 \\sim 10: 00$, but the arrival time is random, and the arrival times of the two buses are independent, with the following pattern:\n\nA passenger arrives at the station at $8: 20$, then the expected waiting time for the bus is $\\qquad$ (to the nearest minute).",
        "answer": "27",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 99825,
        "difficulty": null
      },
      {
        "id": "c4c6fdfc3adcb6fbaf9e8d6fe43873d4dccf96d9e52d7d3836b8e16751bdc688",
        "domain": "numina",
        "question": "16. Mailboxes. In a newly inhabited apartment building, workers installed a block of 80 mailboxes. Inside each mailbox, they placed a ring with a key and a tag with the apartment number, but they did not bother to ensure that each mailbox contained the key to that mailbox. They simply threw the keys into the mailboxes in a random order. The mailboxes, of course, are locked.\n\nThe Absent-Minded Scientist, who recently moved into apartment 37, was delighted to see the mailboxes, but the concierge said she could not get the keys out. One had to go to the management company, which accepts citizens every third Monday of the month from 12:00 to 12:30.\n\nFinding a stick in the trash can (someone had ordered sushi), the Scientist managed to extract a key from mailbox № 37 on the fourth try. He reasoned as follows: if this is the key to my mailbox, then everything is fine. If not, I will open the mailbox to which this key belongs, leave the key in the lock, take out the next key, open the next mailbox, and so on, until I get my key.\n\na) (for 8th grade, 3 points). Prove that the described algorithm is effective, that is, that the Absent-Minded Scientist will find his key.\n\nb) (for 9th grade, 6 points). Find the expected number of residents who will have to poke in the mailboxes with a stick if they all follow the Absent-Minded Scientist's algorithm.",
        "answer": "4.965",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10993,
        "difficulty": null
      },
      {
        "id": "524d146e758fd6385daeef52352bbd9f9305719cd38b0808e016b86e40826704",
        "domain": "numina",
        "question": "11. (15 points) Find the minimum value of $g(x)=\\frac{x^{3}+1}{x^{2}-x}(x>1)$.",
        "answer": "\\sqrt{9+6\\sqrt{3}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109487,
        "difficulty": null
      },
      {
        "id": "b4dc2e17a81c64fffce739f12d36a2d3a1e749dcb7ddf26ab10eb4e166af3181",
        "domain": "numina",
        "question": "A grid with nine cells, as shown in the image, is filled with nine consecutive natural numbers. These numbers are arranged in ascending order from left to right and from top to bottom (i.e., the smallest number is in the top left, and the largest is in the bottom right). The thick broken line divides the grid into two parts. The sum of the numbers in the left part is 100 less than the sum of the numbers in the right part.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_f53d56ddc45593eccca2g-2.jpg?height=377&width=376&top_left_y=477&top_left_x=840)\n\nWhich number is in the middle cell?\n\n(L. Šimůnek)",
        "answer": "94",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76392,
        "difficulty": null
      },
      {
        "id": "f981e747e6cf99f6476960f7169e8b45e5534ffa61a1228cb57e3d61da7b3a1b",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$$\n\\lim _{x \\rightarrow 0}\\left(\\frac{1+x \\cdot 3^{x}}{1+x \\cdot 7^{x}}\\right)^{\\frac{1}{x^2}}\n$$",
        "answer": "\\frac{3}{7}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46979,
        "difficulty": null
      },
      {
        "id": "c26ebe32f2970730ae4f61a8639264ce5b62f0ce9429d5efed4dbf5371e54c8d",
        "domain": "numina",
        "question": "3. At a rock festival, vocalists, guitarists, and drummers met, a total of 121 people. Each vocalist gave a slap on the back of each guitarist, each guitarist gave a slap on the back of each drummer, and each drummer gave a slap on the back of each vocalist. What is the maximum number of slaps on the back that the participants of the festival could have received?",
        "answer": "4880",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20562,
        "difficulty": null
      },
      {
        "id": "3a239c928b2dab63c51c36db4d51768d91aca1945bcc8016e5a1b5e26f269bf1",
        "domain": "numina",
        "question": "11. (20 points) As shown in Figure 1, the ellipse $E: \\frac{x^{2}}{4}+y^{2}=1$ has its right focus at $F$, and the circle $\\odot O: x^{2}+y^{2}=1$ has a tangent line $AB$ that intersects the ellipse $E$ at points $A$ and $B$ ($A$ and $B$ are both on the right side of the $y$-axis). Find the maximum value of the inradius of $\\triangle ABF$.",
        "answer": "\\frac{3-\\sqrt{3}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 123559,
        "difficulty": null
      },
      {
        "id": "ce8d9df7bb168f8d66d445cc8121358d7e73f65bfe4a8fb500cd45de1f2e8031",
        "domain": "numina",
        "question": "11.2. Several married couples came to the New Year's Eve party, each of whom had from 1 to 10 children. Santa Claus chose one child, one mother, and one father from three different families and took them for a ride in his sleigh. It turned out that he had exactly 3630 ways to choose the required trio of people. How many children could there be in total at this party?\n\n(S. Volchonkov)",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 15743,
        "difficulty": null
      },
      {
        "id": "4681de64c15cfd10d657e8a5506ff81dfc330ed8a2a08f3eb95b60f9927a1074",
        "domain": "numina",
        "question": "## Task 16/65\n\nThe difference between the difference of the cubes and the difference of the squares of two consecutive natural numbers is 114. Which numbers are these?",
        "answer": "76",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 182110,
        "difficulty": null
      },
      {
        "id": "94b1efcd4fd8256298c8015fbf49124d8d73c86c5243ffec8f20fd06edbcc6b4",
        "domain": "numina",
        "question": "Podlipsky O.K.\n\nOleg drew an empty $50 \\times 50$ table and wrote a number above each column and to the left of each row. It turned out that all 100 written numbers are distinct, with 50 of them being rational and the other 50 being irrational. Then, in each cell of the table, he wrote the product of the numbers written next to its row and its column (\"multiplication table\"). What is the maximum number of products in this table that could be rational numbers?",
        "answer": "1275",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 52848,
        "difficulty": null
      },
      {
        "id": "2d953f1af45c03fd7cedd1e5a564cfdddd5c27943b096cc39fcc25ea8b49accc",
        "domain": "numina",
        "question": "A tank can be filled with water through two pipes. One pipe can fill the tank in 3 hours, and both pipes together can fill the tank in 1 hour and 12 minutes. How many hours does it take to fill the tank if only the second pipe is used?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 172674,
        "difficulty": null
      },
      {
        "id": "d6ae5e5faa4b167a0ca41314cf4d9581ee61219f8f3f220a3746d2bef61eab3d",
        "domain": "numina",
        "question": "We randomly place points $A, B, C$, and $D$ on the circumference of a circle, independently of each other. What is the probability that the chords $AB$ and $CD$ intersect?",
        "answer": "\\frac{1}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 171751,
        "difficulty": null
      },
      {
        "id": "787027d7cf980d98ec5790582c1109931d06c22f1578c8d2a86e49f4aa1ea9d1",
        "domain": "numina",
        "question": "3. Andrew has two containers for carrying water. The containers are cubes without tops and have base areas of $4 \\mathrm{dm}^{2}$ and $36 \\mathrm{dm}^{2}$ respectively. Andrew has to completely fill the larger cube with pond water, which must be carried from the pond using the smaller cube. What is the smallest number of visits Andrew has to make to the pond with the smaller cube?",
        "answer": "27",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156431,
        "difficulty": null
      },
      {
        "id": "328606da4cdd4a7d1c417ce13ee7f0c1e063e1d55b1d5da8c59923aab0ba36af",
        "domain": "numina",
        "question": "2. Let there be $n$ voters, and $k$ votes were cast for a given candidate. Then $0.50332 \\leqslant$ $\\frac{k}{n} \\leqslant 0.50333$, and $1.00664 \\leqslant \\frac{2 k}{n} \\leqslant 1.00666$. If we denote $m=2 k-n$, then $0.00664 \\leqslant \\frac{m}{n} \\leqslant$ 0.00666.\n\n- If $m=1$, then $150.1<\\frac{1}{0.00666} \\leqslant n \\leqslant \\frac{1}{0.00664}<150.7$ - no integer solutions.\n- If $m=2$, then $300<\\frac{2}{0.00666} \\leqslant n \\leqslant \\frac{2}{0.00664}<302$. But if $n=301$ and $m=2$, then the corresponding value of $k$ does not exist.\n- If $m=3$, then $450<\\frac{3}{0.00666} \\leqslant n \\leqslant \\frac{3}{0.00664}<452$. Therefore, $n=451, k=227$, and all inequalities are satisfied.",
        "answer": "451",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 5281,
        "difficulty": null
      },
      {
        "id": "4b14b62c818a486d21161430f598f6c37a7b8e99936446082d0ce85ec3ea7ece",
        "domain": "numina",
        "question": "Problem 3. How many natural numbers $n>1$ exist, for which there are $n$ consecutive natural numbers, the sum of which is equal to 2016?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9453,
        "difficulty": null
      },
      {
        "id": "69fe97b8ef86360eb348562ebead85d7af6ab19991caa8322f6f81de6ad96703",
        "domain": "numina",
        "question": "Example 4: Given 1 one-yuan note, 1 two-yuan note, 1 five-yuan note, 4 ten-yuan notes, and 2 fifty-yuan notes, how many different amounts of money can be paid using these notes?\n(1986 Shanghai Competition Question)",
        "answer": "119",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 93791,
        "difficulty": null
      },
      {
        "id": "406a8174fec682a56ac4d45a5b95dd4715f35f0e4be168f0658bc9bb7eecb777",
        "domain": "numina",
        "question": "[ Right triangle with an angle in ]\n\nFrom the vertex of the right angle of triangle $ABC$, the median $CM$ is drawn. The circle inscribed in triangle $CAM$ touches $CM$ at its midpoint. Find the angle $BAC$.\n\n#",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67390,
        "difficulty": null
      },
      {
        "id": "b9c663aa4aa6c4c878b4dd54ad725329c0865bc569cc65e7685c8c813c5e0112",
        "domain": "numina",
        "question": "7. In a $100 \\times 100$ square, 10000 cuts were made along the grid lines (each cut one cell long), and it split into 2500 four-cell figures. How many of them are $2 \\times 2$ squares?",
        "answer": "2300",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21454,
        "difficulty": null
      },
      {
        "id": "cf786e486506c19c4eb239a21b59dbb533fb128a09d0e4db24f62c18e14b245f",
        "domain": "numina",
        "question": "1. Let $Q(x)=a_{0}+a_{1} x+\\cdots+a_{n} x^{n}$ be a polynomial with integer coefficients, and $0 \\leq a_{i}<3$ for all $0 \\leq i \\leq n$.\nGiven that $Q(\\sqrt{3})=20+17 \\sqrt{3}$, compute $Q(2)$.",
        "answer": "86",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151635,
        "difficulty": null
      },
      {
        "id": "ee4aa22cd6b2cb8964c36eca68bd6304b9aae59f8166f175703a2cef833c803f",
        "domain": "numina",
        "question": "7.055. $\\log _{3}\\left(3^{x}-8\\right)=2-x$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.\n\n7.055. $\\log _{3}\\left(3^{x}-8\\right)=2-x$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50827,
        "difficulty": null
      },
      {
        "id": "f77da403bed8d12cdb20f2d89d4363327da161fa0574c8371bc3b0af9db814ae",
        "domain": "numina",
        "question": "20th Brazil 1998 Problem A2 ABC is a triangle. D is the midpoint of AB, E is a point on the side BC such that BE = 2 EC and ∠ADC = ∠BAE. Find ∠BAC.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 149181,
        "difficulty": null
      },
      {
        "id": "4df0a202304437cab6fac0008351ad883b1c207508d75e9e4ec4293e075362b8",
        "domain": "numina",
        "question": "13.012. The working day has been reduced from 8 to 7 hours. By what percentage does labor productivity need to increase to ensure that, with the same rates, the wage increases by $5 \\%$?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51520,
        "difficulty": null
      },
      {
        "id": "27359fbcf80ea6eba1462149c085a4c61c57d13df6604b61051d0c2901ce03a1",
        "domain": "numina",
        "question": "7. If $n$ is a positive integer such that $n^{6}+206$ is divisible by $n^{2}+2$, find the sum of all possible values of $n$.\n(1 mark)\n若 $n$ 是正整數, 且 $n^{6}+206$ 可被 $n^{2}+2$ 整除, 求 $n$ 所有可能值之和。\n(1 分)",
        "answer": "32",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153577,
        "difficulty": null
      },
      {
        "id": "3895926f45ef710f69aea74240db42af500798c236c131599d29c51302b59ebd",
        "domain": "numina",
        "question": "Kuuyggin A.K.\n\nAmong any five nodes of a regular square grid, there will always be two nodes such that the midpoint of the segment between them is also a node of the grid. What is the minimum number of nodes of a regular hexagonal grid that must be taken so that among them there will always be two nodes such that the midpoint of the segment between them is also a node of this grid?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 54451,
        "difficulty": null
      },
      {
        "id": "2823bf0d3022fbccd81443503d7768a06dfe1e8e04142b36a8a6ac367a207de5",
        "domain": "numina",
        "question": "## In a convex quadrilateral $A B C D$, the length of the segment connecting the midpoints of sides $A B$ and $C D$ is 1. Lines\n\n$B C$ and $A D$ are perpendicular. Find the length of the segment connecting the midpoints of diagonals $A C$ and $B D$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62100,
        "difficulty": null
      },
      {
        "id": "459b26544ab7cf981356014473b1e6611352acabb8351677124bb8c94a651b6c",
        "domain": "numina",
        "question": "118. Find $\\lim _{\\alpha \\rightarrow 0} \\frac{\\sin 2 \\alpha}{\\alpha}$.",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31940,
        "difficulty": null
      },
      {
        "id": "78f332764fdc0b463731fda14853a30a8182731971fc40661b384ba57f7209af",
        "domain": "numina",
        "question": "4. The rectangles $S_{1}$ and $S_{2}$ are drawn on a set of coordinate axes. $S_{1}$ has vertices at $(0, y) ;(x, y) ;(x, 4)$; and $(0,4)$, where $x<9$ and $y<4$. $S_{2}$ has vertices at $(x, 0)$; $(9,0) ;(9, y)$; and $(x, y)$. The rectangles are equal in area. What is the value of $\\frac{360 x}{y}$ ?",
        "answer": "810",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156352,
        "difficulty": null
      },
      {
        "id": "fc6994dddd2067c260242d77257255cd0746803b827a683a500df7bbcbd43f7b",
        "domain": "numina",
        "question": "6. A circle can be circumscribed around quadrilateral $A B C D$. The lengths of sides $A B$ and $A D$ are equal.\n\nOn side $C D$ is point $Q$ such that $D Q=1$, and on side $B C$ is point $P$ such that $B P=5$.\n\nFurthermore, $\\measuredangle D A B=2 \\measuredangle Q A P$. Find the length of segment $P Q$.\n\n## Solutions and answers\n\n## Variant 1\n\nProblem 1 Answer: $7-\\frac{\\sqrt{2}}{2}$",
        "answer": "7-\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 2212,
        "difficulty": null
      },
      {
        "id": "069ffcd8fb77b676fdf471bcd6eed261ba7cfc89c446b4a13ea8298bb75ad5a5",
        "domain": "numina",
        "question": "Šárka and Terezka received a box of chocolates, containing 35 chocolate candies. Each day, each of the girls ate at least one candy, and no candy was divided into parts. On the first day, Terezka ate $\\frac{2}{5}$ of what Šárka ate on the same day. On the second day, Šárka ate $\\frac{3}{4}$ of what Terezka ate on the same day. By the end of the second day, the box was empty.\n\nHow many candies did Terezka eat in total, given that the difference between the number of candies eaten by Terezka and the number of candies eaten by Šárka was the smallest possible?\n\n(L. Hozová)",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76574,
        "difficulty": null
      },
      {
        "id": "2cc7ab584326e0d06e26fda77436a3a47904b21e52eaa5a25975eb64eba7f0eb",
        "domain": "numina",
        "question": "Task 7. A subscriber has forgotten the last digit of the phone number and therefore dials it at random. Determine the probability that he will need to call no more than three times to get the correct connection.",
        "answer": "0.3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 31401,
        "difficulty": null
      },
      {
        "id": "7902449c1097068a5abf3ec1bb5e02a3ba14feb179eb8a5e62deded9cb689e63",
        "domain": "numina",
        "question": "7.3. In the 7th grade, there are 25 people, and each attends a dance club or a drama club (some attend both clubs). Everyone took a math test, and the teacher calculated the percentage of students who received a failing grade (2) separately among the \"dancers\" and among the \"actors.\" It turned out that the percentage is the same and equals $30 \\%$. Can the percentage of failing grades in the entire class be more than $35 \\%$?",
        "answer": "36",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 22270,
        "difficulty": null
      },
      {
        "id": "187a2d3f79d819152ff3a85a7c2849d3df6bc8f7671a47d3d0a9e524b8e12fcc",
        "domain": "numina",
        "question": "Three, (50 points) In an $8 \\times 8$ grid, each cell contains a knight or a knave. Knights always tell the truth, and knaves always lie. It is known that each resident in the grid says: “The number of knaves in my column is (strictly) greater than the number of knaves in my row.” Determine the number of arrangements of residents that satisfy this condition.",
        "answer": "255",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 144824,
        "difficulty": null
      },
      {
        "id": "dbf995c089752a1f0ecb2bce528d6ee2d47425860c7e6a1305b9aa4a1afd33d0",
        "domain": "numina",
        "question": "[\n\nThe Tale of the Dead Princess and the Seven Bogatyrs. Once, returning home in the evening, the bogatyrs gave the princess their prey - 29 gray ducks. Each brother shot at least one duck. All of them hunted a different number of ducks: the older the brother, the more game he shot. What was the catch of the eldest brother?\n\n#",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 70063,
        "difficulty": null
      },
      {
        "id": "2c8054ecd3a4f1436494a0f8b54312a39869edda87fd5de8f40a0135be8b7743",
        "domain": "numina",
        "question": "## 10. Anniversary\n\n\"Do you remember Leonitina's fortieth birthday? It was December 28, $19 \\ldots$ Since then, Leonitina has aged. I noticed that half of her age equals twice the sum of its digits.\"\n\nPlease fill in the missing digits in the date of Leonitina's fortieth birthday.",
        "answer": "1970",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 42133,
        "difficulty": null
      },
      {
        "id": "45912d59e639c146675df3e18a48c9a647778dc9d6b2d77628651542dced1e3c",
        "domain": "numina",
        "question": "1. A mathematician left point A for point B. After some time, a physicist also left point A for point B. Catching up with the mathematician after 20 km, the physicist, without stopping, continued to point B and turned back. They met again 20 km from B. Then each continued in their respective directions. Upon reaching points A and B respectively, they turned and walked towards each other again. How many kilometers from point A will they meet for the third time, if the distance between points A and B is 100 km?",
        "answer": "45",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18391,
        "difficulty": null
      },
      {
        "id": "6ac3cd690476e124b056aefe8f6e8f493142fd4ebbd5f0915a5752cde3066040",
        "domain": "numina",
        "question": "## 86. Math Puzzle $7 / 72$\n\nThe water tank of the LPG contains $1000 \\mathrm{~m}^{3}$ of water and can be filled by two pumping stations in two days, provided no water is consumed. One pump delivers $300 \\mathrm{~m}^{3}$ more water per day than the other.\n\nHow many days can the LPG withdraw $400 \\mathrm{~m}^{3}$ of water daily if the larger pumping station is out of service?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181772,
        "difficulty": null
      },
      {
        "id": "6c87213f4b1cee94df4637080e196e95b85f141db394ea36b6bf2c4ed7975ef4",
        "domain": "numina",
        "question": "Problem 2. Vanya thought of a two-digit number, then swapped its digits and multiplied the resulting number by itself. The result turned out to be four times larger than the number he thought of. What number did Vanya think of?",
        "answer": "81",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 5807,
        "difficulty": null
      },
      {
        "id": "168b839f00df2ef6fd0303eae00d6fc5df747485dfc5bf55b0441c98947d7247",
        "domain": "numina",
        "question": "Problem 3. Little kids were eating candies. Each one ate 7 candies less than all the others together, but still more than one candy. How many candies were eaten in total?\n\n[5 points] (A. V. Shapovalov)",
        "answer": "21",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22722,
        "difficulty": null
      },
      {
        "id": "f5b275443a8a66d1f9250bf81b79b4ed873d3c746a28a35af4975ef9a126500b",
        "domain": "numina",
        "question": "4. (15 points) The efficiency of an ideal heat engine is $50 \\%$. What will it become if the temperature of the heater is increased by $50 \\%$, and the temperature of the cooler is decreased by $50 \\%$?",
        "answer": "83",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4343,
        "difficulty": null
      },
      {
        "id": "5ee4285eb84830d56ba0e4ca6214bd814bbf90b5bc0b340365fca4ec64692f7a",
        "domain": "numina",
        "question": "1. (5 points) The left figure below is the recently discovered Archimedes' \"Stomachion\" puzzle, dividing a square into 14 polygons: After research, experts found that the 14 pieces can be correctly drawn on a square with a side length of $12 \\mathrm{~cm}$, as shown in the figure. Question: The area of the gray piece is $\\qquad$ square centimeters.\n12.",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 99676,
        "difficulty": null
      },
      {
        "id": "7f1531247951af258bfb5d47615b5a99994ba66e1581a2ebfb0e09fba23a74a0",
        "domain": "numina",
        "question": "Sorrelkova N.P.\n\nUnder one of the cells of an $8 \\times 8$ board, a treasure is buried. Under each of the other cells, there is a sign indicating the minimum number of steps required to reach the treasure from that cell (one step allows moving from a cell to an adjacent cell by side). What is the minimum number of cells that need to be dug up to definitely retrieve the treasure?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 64408,
        "difficulty": null
      },
      {
        "id": "96c5ba1abc46b6f9bd51867610bc43a5bdad6e09224dcbca1f93977a3a9cd117",
        "domain": "numina",
        "question": "10. Find the maximum of the expression\n$$\n|| \\ldots|| x_{1}-x_{2}\\left|-x_{3}\\right|-\\ldots\\left|-x_{2023}\\right|,\n$$\nwhere $x_{1}, x_{2}, \\ldots, x_{2023}$ are distinct natural numbers between 1 and 2023 .",
        "answer": "2022",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153968,
        "difficulty": null
      },
      {
        "id": "e424ca1166fa3b9a1c03bf382f00f64ae71847a40a5d6b48379392b8341fa99c",
        "domain": "numina",
        "question": "Compute the sum $S=\\sum_{i=0}^{101} \\frac{x_{i}^{3}}{1-3 x_{i}+3 x_{i}^{2}}$ for $x_{i}=\\frac{i}{101}$.\n\nAnswer: $S=51$.\n\n#",
        "answer": "51",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91025,
        "difficulty": null
      },
      {
        "id": "5b2731ff53ff2a71897610c1310e19037e751d476fa899eeb6557374287ced23",
        "domain": "numina",
        "question": "Problem 9.5. Positive numbers $a, b$, and $c$ satisfy the equations\n\n$$\na^{2}+a b+b^{2}=1, \\quad b^{2}+b c+c^{2}=3, \\quad c^{2}+c a+a^{2}=4\n$$\n\nFind $a+b+c$.",
        "answer": "\\sqrt{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4445,
        "difficulty": null
      },
      {
        "id": "0fc65ecbac22ee128d8df4ece3c25deaf53f5934c279e107d5ac9cb970c783ae",
        "domain": "numina",
        "question": "## 12. Punctual Wife\n\nEvery Saturday between 15 and 16, I play tennis with my friend Philippe. My wife arrives to pick me up in the car precisely at 16:10. One day, Philippe fell ill. Unaware of this, I set out for tennis as usual. However, at 15:05, realizing that Philippe wasn't coming, I gathered my things and started walking home. On the way, I met my wife, who was driving to pick me up. I got into the car, and we arrived home 10 minutes earlier than usual.\n\nThis might not seem very interesting, but could you determine from this story the ratio between my walking speed and the speed at which my wife drives?",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 42068,
        "difficulty": null
      },
      {
        "id": "330cba82ad1f96d3b2584e8031d0c7ab0a7b1dc3027b25b83aead4cadb0bc5ec",
        "domain": "numina",
        "question": "Analyzing the 4-digit natural numbers:\n\na) How many of them have all different digits?\n\nb) How many have the digit 1 exactly once and all different digits?\n\nc) How many have the digit 1?",
        "answer": "3168",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74004,
        "difficulty": null
      },
      {
        "id": "646e757daf92e2afa40f6cf9496b7367cd5ab648c9a37b862f80a995a2dc399d",
        "domain": "numina",
        "question": "9.1. It is known that the sum of the digits of number A is 59, and the sum of the digits of number B is 77. What is the minimum sum of the digits that the number A+B can have?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9797,
        "difficulty": null
      },
      {
        "id": "bf9e6fe11ec3e80f1f160dfe5068df7a9309444f5972ebd75269e9fa071cfced",
        "domain": "numina",
        "question": "7. Each cell of a $10 \\times 10$ board is painted black or white. A cell is said to be out of place if it has at least seven neighbors of a different color than itself. (Neighbors are cells that share a common side or corner.) What is the maximum number of white cells on the board that can be out of place at the same time?",
        "answer": "26",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21705,
        "difficulty": null
      },
      {
        "id": "d19eaf288adf36428c50867b7e46b227ce17c823b3eca526fd09e5e0ae1dd521",
        "domain": "numina",
        "question": "36. Given that each of the values of $x_{1}, x_{2}, \\cdots \\cdots, x_{2018}$ is either -1 or 1, if $x_{1}+x_{2}+\\cdots+x_{2018}=4$, then the maximum value of $\\left|x_{1}-x_{2}\\right|+\\left|x_{3}-x_{4}\\right|+\\cdots+\\left|x_{2017}-x_{2018}\\right|$ is $\\qquad$",
        "answer": "2014",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 115089,
        "difficulty": null
      },
      {
        "id": "a8dd14230829f0eae7771f05db3fc2e301978c08718b62bee334b73e58440f30",
        "domain": "numina",
        "question": "11. Let $A$ be a set composed of any 100 distinct positive integers, and let\n$$\nB=\\left\\{\\left.\\frac{a}{b} \\right\\rvert\\, a, b \\in A \\text { and } a \\neq b\\right\\},\n$$\n$f(A)$ denotes the number of elements in set $B$. Then the sum of the maximum and minimum values of $f(A)$ is $\\qquad$",
        "answer": "10098",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 141424,
        "difficulty": null
      },
      {
        "id": "c0bc34c69c73a7468a43e3e3680d75fb6ed4c9b6c6b4bdc8ffc2dd2d850c5879",
        "domain": "numina",
        "question": "6. Solve the equation $\\sqrt{\\frac{x-3}{2 x+1}}+2=3 \\sqrt{\\frac{2 x+1}{x-3}}$.",
        "answer": "-4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19140,
        "difficulty": null
      },
      {
        "id": "eb296ff565903ddf70e6907fed94ef653339c9af279913de2d9e5e48d29109ce",
        "domain": "numina",
        "question": "3. (13 points) Money placed in the 1st month will be on deposit for 6 months and will bring a nominal income of $8700\\left((1+0.06 / 12)^{\\wedge}-1\\right)=264.28$ rubles.\n\nFunds placed in the 2nd month will be on deposit for 5 months and will bring a nominal income of $8700\\left((1+0.06 / 12)^{\\wedge} 5-1\\right)=219.69$ rubles.\n\nSimilarly in the following months:\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 4-1\\right)=175.31$ rubles.\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 3-1\\right)=131.15$ rubles.\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 2-1\\right)=87.22$ rubles.\n\n$8700((1+0.06 / 12)-1)=43.5$ rubles.\n\nThus, the nominal income for 6 months will be:\n\n$264.28+219.69+175.31+131.15+87.22+43.5=921.15$ rubles.\n\n## Maximum 25 points\n\n## Problem 5\n\nMobile applications can significantly improve the convenience of paying for services and goods. Imagine that you are developing a mobile internet banking application where users can pay for services of various organizations with their bank card.\n\nList at least 3 security measures that you should implement to protect users of the application from theft of money from their bank card.\n\n#",
        "answer": "921.15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11447,
        "difficulty": null
      },
      {
        "id": "c6cd6665f3175a542952fe484f66e24bb40cf7c849184e66556ba2c0d7a8ce43",
        "domain": "numina",
        "question": "14. From the 20 numbers 11, 12, 13, 14, ... 30, at least ( ) numbers must be taken to ensure that there are two numbers among the taken numbers whose sum is a multiple of ten.",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 129020,
        "difficulty": null
      },
      {
        "id": "e7f08e80bed46613f5cead2b971ee9734c66e92c999ff7e0779373cda28bafd2",
        "domain": "numina",
        "question": "$7 \\cdot 8$ If a positive divisor is randomly selected from $10^{99}$, the probability that it is exactly a multiple of $10^{88}$ is $\\frac{m}{n}$, where $m$ and $n$ are coprime, find $m+n$.\n\nThe prime factorization of $10^{99}$ is $2^{99} \\cdot 5^{99}$. The total number of positive divisors of $10^{99}$ is $(99+1)(99+1) = 100 \\cdot 100 = 10000$.\n\nFor a divisor to be a multiple of $10^{88}$, it must be of the form $2^{a} \\cdot 5^{b}$ where $a \\geq 88$ and $b \\geq 88$. The number of such divisors is $(100-88)(100-88) = 12 \\cdot 12 = 144$.\n\nThus, the probability that a randomly selected divisor of $10^{99}$ is a multiple of $10^{88}$ is $\\frac{144}{10000} = \\frac{9}{625}$. Here, $m = 9$ and $n = 625$, and they are coprime.\n\nTherefore, $m+n = 9 + 625 = 634$.",
        "answer": "634",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 121988,
        "difficulty": null
      },
      {
        "id": "4c590122e317cd4dfd65b0f027243a7cb4b6b936747f2bb7a39b5cb9b1c0a477",
        "domain": "numina",
        "question": "Question 39, Let $a, b$ be real numbers, and the function $f(x)=x^{3}+a x^{2}+b x$. If there exist three real numbers $x_{1}, x_{2}, x_{3}$ satisfying $x_{1}+1 \\leq x_{2} \\leq x_{3}-1$, and $f\\left(x_{1}\\right)=f\\left(x_{2}\\right)=f\\left(x_{3}\\right)$, and the minimum value of $|a|+2|b|$.",
        "answer": "\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100616,
        "difficulty": null
      },
      {
        "id": "36b70bfb50f21a758594c4d63d575aabe5fc0e800cfed3cfa49bcf3743ffa9a4",
        "domain": "numina",
        "question": "8 The function $f(x)=\\frac{x-x^{3}}{1+2 x^{2}+x^{4}}$ has a product of its maximum and minimum values of\n\n保留源文本的换行和格式，翻译结果如下：\n\n8 The function $f(x)=\\frac{x-x^{3}}{1+2 x^{2}+x^{4}}$ has a product of its maximum and minimum values of",
        "answer": "-\\frac{1}{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138213,
        "difficulty": null
      },
      {
        "id": "5ac1b5eed6fa5093c3b55dfa05f2b2aa469d4d265c5820bc3c8a87f1c531291d",
        "domain": "numina",
        "question": "3. The triangle $A B C$ is isosceles with $A B=B C$. The point $D$ is a point on $B C$, between $B$ and $C$, so that $A C=A D=B D$.\nWhat is the size of angle $A B C$ ?",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 154943,
        "difficulty": null
      },
      {
        "id": "ac62761d0b8d06aa0f9722d3810cc29c26e6260582d3ec2ebdc3656f5603dc26",
        "domain": "numina",
        "question": "5.5. The hedgehogs collected 65 mushrooms and divided them so that each hedgehog got at least one mushroom, but no two hedgehogs had the same number of mushrooms. What is the maximum number of hedgehogs that could be",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18236,
        "difficulty": null
      },
      {
        "id": "4ec6d0d25581dc4a1ccdfac26052bf65d48fdb8c2e73900db1f4c914fb7740e9",
        "domain": "numina",
        "question": "15. In the complex plane, the three vertices of $\\triangle A B C$ correspond to the complex numbers $z_{1}, z_{2}, \\dot{z}_{3}$. It is known that $\\left|z_{1}\\right|=3$, $z_{2}=\\bar{z}_{1}, z_{3}=\\frac{1}{z_{1}}$, find the maximum value of the area of $\\triangle A B C$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122946,
        "difficulty": null
      },
      {
        "id": "a0370481321df99c45c869587070772108a9c27436c1ace059237fb09c5c333e",
        "domain": "numina",
        "question": "20、Person A and Person B start from points $A$ and $B$ respectively at the same time, heading towards each other. After they meet, A increases his speed by 10 kilometers per hour, and B decreases his speed by 5 kilometers per hour. When A reaches point B, B is exactly at the midpoint of points $A$ and $B$. If B's original speed was $\\frac{2}{2}$ of A's speed, then A's original speed was $\\qquad$ kilometers per hour.",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136028,
        "difficulty": null
      },
      {
        "id": "27feb1d97a70c6d5ac3048dc252db1bbdca95a74e9c5b5837b3809662f7f2f01",
        "domain": "numina",
        "question": "The kindergarten received cards for reading lessons: some have \"MA\" written on them, and others have \"NYA\". Each child took three cards and started forming words from them. It turned out that 20 children can form the word \"MAMA\", 30 children can form the word \"NYANYA\", and 40 children can form the word \"MANYA\". How many children have all three cards the same?\n\n#",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56791,
        "difficulty": null
      },
      {
        "id": "56c647575f36126b7a06a3eb75056707c2e24b38f4b18b41e609b10cca41a3be",
        "domain": "numina",
        "question": "3. Find the smallest trimestral number for which all the digits of its triple are even.",
        "answer": "134",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162828,
        "difficulty": null
      },
      {
        "id": "b5330ca0ea05ccae49d2c8ab9edf3f2986c3bff72beaef9b973ba9fa5b5e0787",
        "domain": "numina",
        "question": "6. On the table, there are three spheres and a cone (with its base $k$ on the table), touching each other externally. The radii of the spheres are 20, 40, and 40, and the radius of the base of the cone is 21. Find the height of the cone.",
        "answer": "28",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8194,
        "difficulty": null
      },
      {
        "id": "e8b4271a258a52fe24a18cee82b4fa0719ace2a5830a92955ced9a891b861b3a",
        "domain": "numina",
        "question": "4. All the 7-digit numbers containing each of the digits $1,2,3,4,5,6,7$ exactly once, and not divisible by 5 , are arranged in the increasing order. Find the 2000-th number in this list.",
        "answer": "4315672",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 180398,
        "difficulty": null
      },
      {
        "id": "d7ba3a1448c18d1797b8b3110180c7bc4f3319cdbca005a437c1e03eca5bf22e",
        "domain": "numina",
        "question": "2・102 Let the set $S=\\{1,2, \\cdots, 50\\}$, find the smallest positive integer $n$, such that in every $n$-element subset of $S$ there are 3 numbers that can serve as the 3 side lengths of a right triangle.",
        "answer": "42",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 140257,
        "difficulty": null
      },
      {
        "id": "e9590ab68e72071477c7fc351d9915748c9806274a2299790f84ef1b43156ecd",
        "domain": "numina",
        "question": "7. Bruno set a four-digit even number as the unlock code for his mobile phone. To avoid forgetting the number, he wrote down the following:\n\n- All digits are different, and the sum of all digits is 15.\n- The unit digit is three times smaller than the thousands digit.\n- The tens digit is smaller than the unit digit.\n\nDetermine the code that Bruno set for unlocking his mobile phone.",
        "answer": "6702",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 86910,
        "difficulty": null
      },
      {
        "id": "ebca9545e6cd7d4fffddaabe1e487d36192f0eb315a1d2c9f3e9136139471d12",
        "domain": "numina",
        "question": "3. (8 points) The largest four-digit number is $\\qquad$ times larger than the largest two-digit number.",
        "answer": "100",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124430,
        "difficulty": null
      },
      {
        "id": "2533676824a5adbb9bd9dc7a921786fe57ffbcfba6d02e27ab8f12b4727e1a4b",
        "domain": "numina",
        "question": "The focus of the parabola $y^{2}=7 x$ is crossed by a line which intersects the $Y$ axis at (-1). What is the area of the parabolic segment thus obtained?",
        "answer": "66.87",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173525,
        "difficulty": null
      },
      {
        "id": "c8cb1dbbc48e9abdc2481bac7989325bc241028d01ace20bafe733df06546824",
        "domain": "numina",
        "question": "# Problem 9.\n\nLet $A(n)$ denote the greatest odd divisor of the number $n$. For example, $A(21)=21$, $A(72)=9, A(64)=1$. Find the sum $A(111)+A(112)+\\ldots+A(218)+A(219)$.",
        "answer": "12045",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6319,
        "difficulty": null
      },
      {
        "id": "9ed8cda0622fe944a502ca1531dc18cef2d28e3dc870fa66f2262221908bedb4",
        "domain": "numina",
        "question": "15. English Club (from 9th grade. 6 points). Every Friday, ten gentlemen come to the club, and each hands their hat to the doorman. Each hat fits its owner perfectly, but no two hats are the same size. The gentlemen leave one by one in a random order.\n\nWhen seeing off each gentleman, the doorman tries to put on the first hat he finds. If it fits, the gentleman leaves with that hat. If it is too small, the doorman tries the next random hat from the remaining ones. If all the remaining hats are too small, the doorman says to the poor man: \"Sir, a hat does not suit you today,\" and the gentleman goes home with an uncovered head. Find the probability that next Friday the doorman will not have any hats left.\n\nIX Olympiad in Probability Theory and Statistics for Schoolchildren. Correspondence Round.\n\n© Laboratory of Probability Theory and Statistics of the MCNMO, 2015-2016\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_be2097b0254fa1f5b26ag-18.jpg?height=391&width=534&top_left_y=216&top_left_x=293)",
        "answer": "0.000516",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 11033,
        "difficulty": null
      },
      {
        "id": "7d3298a486d85364744500beeae9cb965d0c08efce9ee41e541e0ec26749975e",
        "domain": "numina",
        "question": "16. Given the function $f(x)=\\left(x^{2}-4\\right)^{-\\frac{1}{2}}(x>2), f^{-1}(x)$ is the inverse function of $f(x)$.\n(1) If the sequence $\\left\\{a_{n}\\right\\}, a_{1}=1, \\frac{1}{a_{n+1}}=f^{-1}\\left(a_{n}\\right)\\left(n \\in \\mathbf{N}_{+}\\right)$, find $a_{n}$;\n(2) Let $S_{n}=a_{1}^{2}+a_{2}^{2}+\\cdots+a_{n}^{2}, b_{n}=S_{2 n+1}-S_{n}$, does there exist a largest positive integer $p$, such that for all $n \\in \\mathbf{N}_{+}$, $b_{n}<\\frac{1}{p}$ holds? Please provide an explanation or proof.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129737,
        "difficulty": null
      },
      {
        "id": "7417e144ebb5dd69656643b991e435b04ec150aa482149b8415653bbeb2344ee",
        "domain": "numina",
        "question": "13. (12 points) In a cage, there are chickens and rabbits, with a total of 28 heads. The total number of rabbit feet is 12 more than 3 times the total number of chicken feet. How many rabbits are there in the cage?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110338,
        "difficulty": null
      },
      {
        "id": "fc19aa8bf753c7d3fd44704f0d952f22a2b79965309ab2e9ccb304b0ec061caa",
        "domain": "numina",
        "question": "$13 \\cdot 17$ There is a chain made up of 60 rings, each weighing 1 gram. What is the minimum number of rings that need to be cut so that the segments of the chain can be used to make up weights of 1 gram, 2 grams, $\\cdots, 59$ grams, 60 grams (the cut rings still weigh 1 gram)?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 116438,
        "difficulty": null
      },
      {
        "id": "48f88b22452c2737429b45a51e395fdf66fba4f8db3193daec66fe3fd18a686a",
        "domain": "numina",
        "question": "Problem 1. In two tubs, there are $4 \\frac{1}{6}$ and $3 \\frac{1}{2}$ liters of water respectively. How much water needs to be transferred from the first tub to the second tub so that after the transfer, both tubs have the same amount of water? Explain your answer!",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188729,
        "difficulty": null
      },
      {
        "id": "247229c6585214c371d9563210df88b118c1d620a2955c0de0c5f6bce66f7d5e",
        "domain": "numina",
        "question": "Problem 4.3. How many rectangles exist on this picture with sides running along the grid lines? (A square is also a rectangle.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-06.jpg?height=173&width=206&top_left_y=614&top_left_x=624)",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18105,
        "difficulty": null
      },
      {
        "id": "09e3202e3349b4e8a8e8c90d9c70a47fb2582e0b63acb034f249c04801e640b2",
        "domain": "numina",
        "question": "Example 1 Connect any two vertices of a cube with a straight line, among these lines, the number of pairs of mutually perpendicular skew lines is $\\qquad$.",
        "answer": "78",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 132569,
        "difficulty": null
      },
      {
        "id": "0c47f19f389b0605f1f100962a2df0f1c5c26ac75ed646be7e16c6fbf6645781",
        "domain": "numina",
        "question": "Example 4 As shown in Figure 4, in $\\triangle A B C$, $A B=A C, \\angle A$ $=100^{\\circ}, I$ is the incenter, and $D$ is a point on side $A B$ such that $B D=B I$. Find the measure of $\\angle B C D$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 120831,
        "difficulty": null
      },
      {
        "id": "800f714eaba7d56075d77748294529dca5e6c475e02afc487e4342b1fd3b4f37",
        "domain": "numina",
        "question": "9.8. A straight rod 2 meters long was sawn into $N$ sticks, the length of each of which is expressed as an integer number of centimeters. For what smallest $N$ can it be guaranteed that, using all the resulting sticks, one can, without breaking them, form the contour of some rectangle?\n\n(A. Magazinov)",
        "answer": "102",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12931,
        "difficulty": null
      },
      {
        "id": "6c4a0c043bccb1f1cf32972afc95fb0e1a7ab2460bbbf2e29cb36efd4bbd1e5e",
        "domain": "numina",
        "question": "Problem 5.8. In the country, there are 100 cities: 30 of them are located in the mountainous part of the country, and 70 are in the plain. Over three years, air routes were established between the cities. Each year, 50 new air routes were opened in the country: all cities were randomly divided into 50 pairs, and a route was opened between the cities in the same pair. After three years, it turned out that out of the 150 routes opened, exactly 21 connect pairs of \"mountainous\" cities. How many routes connect pairs of \"plain\" cities?",
        "answer": "81",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13993,
        "difficulty": null
      },
      {
        "id": "0e063e6beec0eff2595e982899af6dfdc3676fe8c6a14eedbf4ea57c541ce705",
        "domain": "numina",
        "question": "You are out walking and see a group of rhinoceroses (which each have two horns and four legs) and triceratopses (which each have three horns and four legs). If you count 31 horns and 48 legs, how many triceratopses are there?",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 90399,
        "difficulty": null
      },
      {
        "id": "b920eb085bfd18ba13ae466cd19357214f9b45f6ec876b9670c68a4c1f8ef1b6",
        "domain": "numina",
        "question": "29. Given that the real numbers $x, y$ and $z$ satisfies the condition $x+y+z=3$, find the maximum value of $f(x y z)=\\sqrt{2 x+13}+\\sqrt[3]{3 y+5}+\\sqrt[4]{8 z+12}$.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160831,
        "difficulty": null
      },
      {
        "id": "c8fb6ca2fdd68955b854c6d7d2a3b9bcc5dac0b06f090a8af2d9db9201a6e5b8",
        "domain": "numina",
        "question": "247. $\\int_{0}^{2} 3 x^{4} d x . \\quad$ 248. $\\int_{-1}^{\\sqrt{3}} 4 t^{3} d t . \\quad$ 249. $\\int_{0}^{1} \\sqrt[3]{x} d x$.",
        "answer": "\\frac{3}{4}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32167,
        "difficulty": null
      },
      {
        "id": "25f5e3be2050bc06d3a2fc7afa22f8ea393562e0dc88d575818c3f01970e1058",
        "domain": "numina",
        "question": "4. Three people want to travel from city A to city B, which is 45 kilometers away from A. They have two bicycles. The speed of a cyclist is 15 km/h, and the speed of a pedestrian is 5 km/h. What is the minimum time they can reach B, if the bicycle can be left unattended on the road?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 11775,
        "difficulty": null
      },
      {
        "id": "bd10333c0d4183d05dfd8e581e74636f526078dd483273549d9288f16e03cb87",
        "domain": "numina",
        "question": "6. In a $4 \\times 4$ square, cells are colored in several colors such that in any $1 \\times 3$ rectangle, there are two cells of the same color. What is the maximum number of colors that can be used?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21545,
        "difficulty": null
      },
      {
        "id": "8a30fbe88888e1d8ca2369007c30621abd02d0a236bdd4d979c0d2b912c5fb4f",
        "domain": "numina",
        "question": "Standa and Jana received two three-digit numbers. Standa added a decimal point after the first digit in the first number, and after the second digit in the second number, then added the resulting decimal numbers and got the result 50.13. Jana added a decimal point after the second digit in the first number, and after the first digit in the second number, then added the resulting decimal numbers and got the result 34.02.\n\nDetermine the sum of the original three-digit numbers.\n\n(K. Pazourek)",
        "answer": "765",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76282,
        "difficulty": null
      },
      {
        "id": "f69221b01949e07d5895e40a7d6652b8185ef39fff98f748179772b903149093",
        "domain": "numina",
        "question": "We know about a convex quadrilateral that if we divide it with diagonals, the area of three of the resulting four triangles is either $24 \\mathrm{~cm}^{2}$ or $25 \\mathrm{~cm}^{2}$. What is the maximum possible area of the quadrilateral?",
        "answer": "100\\frac{1}{24}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166241,
        "difficulty": null
      },
      {
        "id": "24d9c7ed070d34931cb4fe25cd0b2f91a1326f21808ff365413982d367ef76e9",
        "domain": "numina",
        "question": "Someone wrote down the integers from 1 to 1999 next to each other. What digit stands at the 1999th position?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169754,
        "difficulty": null
      },
      {
        "id": "edea7b1ccfff5e64710517ac751e076c6ca53cf91695a15554b3e773d0b98051",
        "domain": "numina",
        "question": "3. Find the value of $\\frac{1}{1+1^{2}+1^{4}}+\\frac{2}{1+2^{2}+2^{4}}+\\frac{3}{1+3^{2}+3^{4}}+\\cdots+\\frac{100}{1+100^{2}+100^{4}}$.\n(1 mark)\nFind the value of $\\frac{1}{1+1^{2}+1^{4}}+\\frac{2}{1+2^{2}+2^{4}}+\\frac{3}{1+3^{2}+3^{4}}+\\cdots+\\frac{100}{1+100^{2}+100^{4}}$.",
        "answer": "\\frac{5050}{10101}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153508,
        "difficulty": null
      },
      {
        "id": "548207d6c7693e3737e263c3750ce61ada3cdad7d0f5bf6a3bbffd13eaba4a26",
        "domain": "numina",
        "question": "41. A class of natural numbers, starting from the third digit, each digit is exactly the sum of the two preceding digits, such as 134, 279, etc. The number of such three-digit numbers is $\\qquad$.",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 146751,
        "difficulty": null
      },
      {
        "id": "d0c793ee37d53ce6ec00bb9d68393ca8a7fbc546fddf0e697ab78b54bf08f1b0",
        "domain": "numina",
        "question": "【Question 14】\nAs shown in the figure, in rectangle $A B C D$, $A B=6, B C=8$. After folding rectangle $A B C D$ along $C E$, point $D$ lands on point $F$ on diagonal $A C$. What is the area of trapezoid $A B C E$? $\\qquad$",
        "answer": "39",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 91631,
        "difficulty": null
      },
      {
        "id": "f5a827d719081c8c5d8fbf7142d1d1a2e2bb56940d83d91a3173f8df29c5356b",
        "domain": "numina",
        "question": "10. $A 、 B$ two people bought the same number of sheets of letter paper and the same number of envelopes. $A$ put 1 sheet of letter paper in each envelope, and after using up all the envelopes, 40 sheets of letter paper were left; $B$ put 3 sheets of letter paper in each envelope, and after using up all the sheets of letter paper, 40 envelopes were left. They each bought $\\qquad$ sheets of letter paper.",
        "answer": "120",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105251,
        "difficulty": null
      },
      {
        "id": "d844a0dd00f308ae8af103f4e8ec789835b13ade836dbb34b2bd82ab038fe7bb",
        "domain": "numina",
        "question": "18. The diagram shows a semicircle with diameter $P Q$ inscribed in a rhombus $A B C D$. The rhombus is tangent to the arc of the semicircle in two places. Points $P$ and $Q$ lie on sides $B C$ and $C D$ of the rhombus respectively. The line of symmetry of the semicircle is coincident with the diagonal $A C$ of the rhombus. It is given that $\\angle C B A=60^{\\circ}$. The semicircle has radius 10 . The area of the rhombus can be written in the form $a \\sqrt{b}$ where $a$ and $b$ are integers and $b$ is prime. What is the value of $a b+a+b ?$",
        "answer": "603",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156448,
        "difficulty": null
      },
      {
        "id": "1e89e2479f45150daee50265bc517b030e2ad1ac2e06f4a7bd2068fb5a2c5a6d",
        "domain": "numina",
        "question": "5. Given positive numbers $a$ and $b$ satisfy $a+3b=7$. Then the minimum value of $\\frac{1}{1+a}$ $+\\frac{4}{2+b}$ is $\\qquad$ .",
        "answer": "\\frac{13+4\\sqrt{3}}{14}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142506,
        "difficulty": null
      },
      {
        "id": "a4239666a7386cea5df5863744a74d61d09f7e49759ca30e25c0ef22c730b5a0",
        "domain": "numina",
        "question": "3. Felix chooses a positive integer as the starting number and writes it on the board. He then repeatedly performs the following step: he replaces the number $n$ on the board with $\\frac{1}{2} n$ if $n$ is even and with $n^{2}+3$ if $n$ is odd.\n\n(a) Prove that if Felix starts with an odd number, the next two numbers he writes will be even.\n\n(b) For how many choices of starting numbers less than 2023 will Felix never write a number with more than four digits on the board?",
        "answer": "21",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85701,
        "difficulty": null
      },
      {
        "id": "d39f9565b5b0d5527cfcc2f52a252f799a2ed72bebe46d360f6c897c4c762610",
        "domain": "numina",
        "question": "12. (5 points) Two people, A and B, start from points $A$ and $B$, which are 10 kilometers apart, and walk towards each other. If they start at the same time, they will meet 1 kilometer away from the midpoint of $A$ and $B$. If A starts 5 minutes later, they will meet at the midpoint of $A$ and $B$. At this time, A has walked $\\qquad$ minutes.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123330,
        "difficulty": null
      },
      {
        "id": "188b8b6b7a40a304555104b66f82aaed0254d37aecddc5876a5f1d4dfeced613",
        "domain": "numina",
        "question": "4. In a charity donation event, a total of 72 people donated, and the total donation amount is a five-digit number, with the ten-thousands digit being 5, the hundreds digit being 3, the tens digit being 7, and the average donation amount per person is an integer. Therefore, the average donation per person is $\\qquad$ yuan.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\n---\n\nNote: The last sentence is a repetition of the instruction and should not be included in the translated text. Here is the corrected version:\n\n4. In a charity donation event, a total of 72 people donated, and the total donation amount is a five-digit number, with the ten-thousands digit being 5, the hundreds digit being 3, the tens digit being 7, and the average donation amount per person is an integer. Therefore, the average donation per person is $\\qquad$ yuan.",
        "answer": "783",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136398,
        "difficulty": null
      },
      {
        "id": "87b8a86e2903ddd426cba6c908efd00e76f0bf1a03dd38228673899c78de970a",
        "domain": "numina",
        "question": "15. Evaluate\n$$\n\\frac{678+690+702+714+\\cdots+1998+2010}{3+9+15+21+\\cdots+327+333} .\n$$",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160553,
        "difficulty": null
      },
      {
        "id": "4f25b1a1e2dcf32a31d20b6f200ecbf16ab5480b2d290691c827778ceecd5cfd",
        "domain": "numina",
        "question": "1. (5 points) Calculate: $(2.016+201) \\times 201.7-20.16 \\times(20.17+2010)=$",
        "answer": "20.1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117508,
        "difficulty": null
      },
      {
        "id": "9d380046bd90be4b0c94bff9a43833c328151ec2eb340c3aa5a5a99f7d2df929",
        "domain": "numina",
        "question": "8.3. Toshi is traveling from point A to point B via point C. From A to C, Toshi travels at an average speed of 75 km/h, and from C to B, Toshi travels at an average speed of 145 km/h. The entire journey from A to B took Toshi 4 hours and 48 minutes. The next day, Toshi travels back at an average speed of 100 km/h. The journey from B to C took Toshi 2 hours, and the journey from C to A was completed at an average speed of 70 km/h. Find the distance between B and C.",
        "answer": "290",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16848,
        "difficulty": null
      },
      {
        "id": "9b2c072c54748e95bc3dc8da183285a28d25d549e1af8ef7a1610d6a11ef8a1a",
        "domain": "numina",
        "question": "Let $a=\\log _{4} 9$ and $b=108 \\log _{3} 8$. What is the integer equal to $\\sqrt{a b}$ ?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194469,
        "difficulty": null
      },
      {
        "id": "fd682828ec92a244fb636a2b734afb48ddb115ba1842e192ade8632610fb699e",
        "domain": "numina",
        "question": "Task 2. What is the smallest natural number that is divisible by 2022 and whose notation starts with 2023?",
        "answer": "20230110",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19277,
        "difficulty": null
      },
      {
        "id": "51054c02fc4a00546a54da40a935d1cc4bdd8887e53459b313841f19af2d01ad",
        "domain": "numina",
        "question": "4. a) On a circle, the natural numbers from 1 to 20 are written in order in a clockwise direction. By a step, we mean to swap the positions of any two of the 20 numbers written on the circle. What is the minimum number of steps required to rearrange these numbers successively in the counterclockwise direction?\n\nb) What is the answer to the question in a) if 21 numbers are written on the circle in a clockwise direction from 1 to 21?\n\n## MATHEMATICS OLYMPIAD  SECTOR STAGE, 21.02.2016  5th GRADE  SOLUTIONS AND ORIENTATIVE SCORING GUIDELINES\n\nNote: Each problem is scored from 0 to 7 points. Only integer scores are given. Any other solution is assimilated according to the scoring guidelines.",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 178780,
        "difficulty": null
      },
      {
        "id": "61edb570b0aa102c9b9a628c51c013117657c6794698bf1f735534bee8c9154b",
        "domain": "numina",
        "question": "Problem 11.2. On each side of a triangle with angles $30^{\\circ}, 60^{\\circ}$ and $90^{\\circ}$ and hypotenuse 1 is chosen a point such that the triangle formed from these three points is a right triangle. What is the least value of the hypotenuse of this triangle?",
        "answer": "\\frac{\\sqrt{39}-\\sqrt{3}}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 177299,
        "difficulty": null
      },
      {
        "id": "9b7b560976ea07f3a1c89653e4f3de8305499df0fafbcc8f57bb2740b996491d",
        "domain": "numina",
        "question": "14th USAMO 1985 Problem 3 A tetrahedron has at most one edge longer than 1. What is the maximum total length of its edges? Solution",
        "answer": "5+\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147629,
        "difficulty": null
      },
      {
        "id": "110ec5076c43743af916e2508f90e35ea5c261e93be3c7a86669f2e7115501e7",
        "domain": "numina",
        "question": "A square area of size $100 \\times 100$ is paved with $1 \\times 1$ square tiles of four colors: white, red, black, and gray - such that no two tiles of the same color touch each other (that is, they do not share a side or a vertex). How many red tiles can there be?",
        "answer": "2500",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 54069,
        "difficulty": null
      },
      {
        "id": "5799f9ddb3788f6040fa1744547d6bf387550421ba82cefdd2b681b632f1e06a",
        "domain": "numina",
        "question": "9. Given that the largest real root of the equation $x^{3}-7 x^{2}+1=0$ is $t$, then the remainder when $\\left[t^{20100}\\right]$ is divided by 7",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117471,
        "difficulty": null
      },
      {
        "id": "e6f01199e17c5222548603ae8c158d4dee818b484f911589dfd4bfe5dcf3ef9c",
        "domain": "numina",
        "question": "6. The integer part and the fractional part of $(\\sqrt{5}+2)^{2 n+1}(n \\in \\mathbf{N})$ are denoted as $P$ and $Q$, respectively, then $Q(P+Q)=$\n$\\qquad$",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122793,
        "difficulty": null
      },
      {
        "id": "90132257069671e4b8fdf2788bc26cfd9f5f0d2c5d47c95d6fe4ddc40643dd2d",
        "domain": "numina",
        "question": "[ The product of the lengths of the segments of chords and the lengths of the segments of secants ]\n\nFrom point $A$, two rays are drawn intersecting a given circle: one at points $B$ and $C$, and the other at points $D$ and $E$. It is known that $A B=7, B C=7, A D=10$. Find $D E$.\n\n#",
        "answer": "\\frac{1}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59672,
        "difficulty": null
      },
      {
        "id": "96a80bcb09a3c37f2587eb10b5bfec4d14c8af6e03310d0290a46b51a1767aa1",
        "domain": "numina",
        "question": "If there exists a positive integer $m$ such that $m!$ ends with exactly $n$ zeros, then the positive integer $n$ is called a \"factorial tail\". How many numbers less than 1992 are not \"factorial tails\"?",
        "answer": "396",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113310,
        "difficulty": null
      },
      {
        "id": "dc056dc057073bd27d58bca6db1525b90761284de121df0946bc457e89cb6ab5",
        "domain": "numina",
        "question": "70. One year, the ages of Xiao Ming's family of four were: Father 30 years old. Mother is two years younger than Father, Brother's age is three times that of Xiao Ming, this year the total age of the four family members is 114, then Xiao Ming's age this year is $\\qquad$",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95150,
        "difficulty": null
      },
      {
        "id": "f633c4accc73e3d270f20ac51fe7fea6e27645be88ffd8599c43b459b358c10c",
        "domain": "numina",
        "question": "【Example 2】 6 athletes participate in the final, with no ties for any positions. Find the number of possible outcomes where Athlete No. 1 does not come in sixth place.",
        "answer": "600",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123278,
        "difficulty": null
      },
      {
        "id": "e89ec5a0d8b8ad3aafe3ccbcffc9a8c9f6613faf525fb5399c033a6b08257126",
        "domain": "numina",
        "question": "10. (20 points) Let $x_{1}, x_{2}, x_{3}$ be non-negative real numbers, satisfying $x_{1}+x_{2}+x_{3}=1$. Find the minimum and maximum values of\n$$\n\\left(x_{1}+3 x_{2}+5 x_{3}\\right)\\left(x_{1}+\\frac{x_{2}}{3}+\\frac{x_{3}}{5}\\right)\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109656,
        "difficulty": null
      },
      {
        "id": "43d1ddc03064e4e1ff7babc8b5c6c555c77ecdb25397e95d44365df1725a7ccd",
        "domain": "numina",
        "question": "3. The first railway independently designed and constructed in our country, the \"Jingzhang Railway,\" adopted a \"human-shaped\" design. Now, a section of the human-shaped road is to be built, as shown in the figure, where the length of $\\mathrm{A} \\rightarrow \\mathrm{C}$ is 25 kilometers, the length of $\\mathrm{C} \\rightarrow \\mathrm{D}$ is 36 kilometers, and the length of $\\mathrm{A} \\rightarrow \\mathrm{B} \\rightarrow \\mathrm{D}$ is 43 kilometers. What is the length of $\\mathrm{B} \\rightarrow \\mathrm{C}$? $\\qquad$ kilometers.",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113883,
        "difficulty": null
      },
      {
        "id": "63cb860550f7c81dfc81fbbef27588e4568fbbf8b8f743b1ad0fff7261accdbc",
        "domain": "numina",
        "question": "2. Which three-digit numbers are more numerous: those in which all digits have the same parity, or those in which adjacent digits have different parity",
        "answer": "225",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16245,
        "difficulty": null
      },
      {
        "id": "55bdf320bc557aba8d5fe8481a32981b9ff5ca347582eedad6a8c94846d1a239",
        "domain": "numina",
        "question": "## Task B-1.6.\n\nLuka and Pero decided to visit a friend and set off from place $A$ to place $B$, which is $170 \\mathrm{~km}$ away. Luka started walking, while Pero was driven by a motorcyclist. After some time, the motorcyclist returned to pick up Luka, and Pero continued on foot, so that both arrived at place $B$ at the same time. Both pedestrians walk at the same speed of $5 \\mathrm{~km} / \\mathrm{h}$, and the motorcyclist drives at a speed of $85 \\mathrm{~km} / \\mathrm{h}$. At what distance from place $B$ and after how much time did the motorcyclist stop and return to pick up Luka? How long did it take for Luka and Pero to travel from place $A$ to place $B$?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88615,
        "difficulty": null
      },
      {
        "id": "b0a0b7fef472d33542156b29ffdff7773ab278ad7aa905920a890f3cdb23d649",
        "domain": "numina",
        "question": "11. (12 points) The three brothers, Big Mao, Second Mao, and Third Mao, Big Mao said to Third Mao: “When dad was 36 years old, my age was 4 times yours, and Second Mao's age was 3 times yours.” Second Mao said: “Yes, at that time, the sum of our ages was exactly half of dad's current age.” Third Mao said: “Now, the total age of the four of us is already 108 years old.” So, how old is Third Mao this year? $\\qquad$",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116598,
        "difficulty": null
      },
      {
        "id": "2f911f243b1142a04325406654d97ca382a48d2bc3361ae1d418de1960e2f67c",
        "domain": "numina",
        "question": "## Zadatak B-1.3.\n\nAko je $\\frac{a}{b}=\\frac{c}{d}, b \\neq 0, d \\neq 0$ koliko je\n\n$$\n\\frac{(a+c)(b+d)}{a+b+c+d}-\\frac{a b}{a+b}-\\frac{c d}{c+d} ?\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88612,
        "difficulty": null
      },
      {
        "id": "dc0a164f136dce192eca6d30f5f5ca92cf8ed5edda2329ec33ab1a927834dcb8",
        "domain": "numina",
        "question": "## Task 1 - 050521\n\nFrom 36 equally large squares, a rectangle is to be formed by placing them together.\n\na) How many solution possibilities are there? (In each solution, all squares should be used.)\n\nb) Which of the possible rectangles has the smallest perimeter?",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 182939,
        "difficulty": null
      },
      {
        "id": "7521c17967df822aeef7e99f8a85bd69a12f5be7a07e786852eaae18acd7c1d1",
        "domain": "numina",
        "question": "Tokarev S.i.\n\nThe set of five-digit numbers $\\{N_1, \\dots, N_k\\}$ is such that any five-digit number, all of whose digits are in increasing order, coincides in at least one digit with at least one of the numbers $N_1, \\dots, N_k$. Find the smallest possible value of $k$.",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 71542,
        "difficulty": null
      },
      {
        "id": "689f999ae3d5111ef1eb8c4a576e00013a886dd6943e6759be346d419730bc13",
        "domain": "numina",
        "question": "Example 5 Given: $a, b, c$ are all positive numbers, find the maximum value of $y=\\frac{a b+2 b c}{a^{2}+b^{2}+c^{2}}$.\nAnalysis In the process of using the arithmetic mean inequality, note that the coefficient ratio of $a b$ and $b c$ should be $1: 2$, so we use parameters to determine the method of splitting $b^{2}$.",
        "answer": "\\frac{\\sqrt{5}}{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 103637,
        "difficulty": null
      },
      {
        "id": "06ad88e66ba1d7b0ec992662ec0005c2a0b5401a7cac216eea6d7aa9cd3ac0b3",
        "domain": "numina",
        "question": "Example 2. The probability density of a random variable $X$ is given by the function\n\n$$\np(x)= \\begin{cases}0 & \\text { if } x \\leq 0 \\\\ x / 2 & \\text { if } 0 < x \\leq 2\\end{cases}\n$$\n\nFind the probability that in the experiment, the variable $X$ will take a value from the interval $(1,2)$.",
        "answer": "0.75",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33000,
        "difficulty": null
      },
      {
        "id": "c979bd8b8d0a481347978db4f72a86ce470318b4c9d6eeafa4d34bc829ba3cc2",
        "domain": "numina",
        "question": "Problem 2. Duorites are creatures that have two horns, while heptorites have 7 horns. In one herd, there were both types of creatures. Together, they had 16 horns. How many duorites and heptorites were there in this herd?",
        "answer": "1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 188853,
        "difficulty": null
      },
      {
        "id": "2bc3d42ca32b5f565121594c5d8fc0314ef9bd4df52e1c9315ce8a3607bd1055",
        "domain": "numina",
        "question": "Sure, here is the translated text:\n\n```\nFind all real numbers $p$ such that the cubic equation\n$\n5 x^{3}-5(p+1) x^{2}+(71 p-1) x+1=66 p\n$\n\nhas three roots that are all natural numbers.\n```",
        "answer": "76",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128560,
        "difficulty": null
      },
      {
        "id": "e653c44a0179e708df127fd9f8eb4057c55c52d9558eaef4c05946c613706f7c",
        "domain": "numina",
        "question": "5. The parabolas in the figure are obtained by shifting the parabola $f(x)=x^{2}$ and are arranged such that the points of their intersections with the $O X$ axis respectively coincide in pairs, and all vertices lie on the same straight line. There are a total of 2020 parabolas. The length of the segment on the $O X$ axis enclosed between the roots of the first parabola is 1; the length of the segment enclosed between the roots of the second parabola is 2. Find the length of the segment on the $O X$ axis enclosed between the roots of the last parabola.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_eefe83f285a2477dfcd6g-2.jpg?height=357&width=539&top_left_y=1689&top_left_x=770)",
        "answer": "2020",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12040,
        "difficulty": null
      },
      {
        "id": "8c4c6d53c4b706d5e0503e436d8d9f095b38095995803e389d7755f27696de0c",
        "domain": "numina",
        "question": "10.4. Find the maximum value of the expression $a+b+c+d-ab-bc-cd-da$, if each of the numbers $a, b, c$ and $d$ belongs to the interval $[0 ; 1]$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15684,
        "difficulty": null
      },
      {
        "id": "d0310f82edb09c20e0586dd00c2a8ba6ec64509b97f3f1a5b11c0f965ab6ad93",
        "domain": "numina",
        "question": "9.1. A four-digit number $n$ was doubled and 1000 was added to the result. As a result, a four-digit number was obtained, written with the same digits as $n$, but in reverse order. Find all possible values of $n$.",
        "answer": "2996",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14796,
        "difficulty": null
      },
      {
        "id": "98a68c245206851cc032ae6fc7c429d87cd9737d393edea966493cf32e4436de",
        "domain": "numina",
        "question": "2. Andrey, Boris, Vasily, Gennady, and Dmitry played table tennis in pairs such that every two of them played with every other pair exactly once. There were no draws in the tennis matches. It is known that Andrey lost exactly 12 times, and Boris lost exactly 6 times. How many times did Gennady win?\n\nOm vem: Gennady won 8 times.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 15975,
        "difficulty": null
      },
      {
        "id": "1ff59b3e147d933c7964203a9b38bfb8af9763603d2761b4430284c444b6114f",
        "domain": "numina",
        "question": "6. Fill in the blanks with appropriate numbers to make the vertical calculation valid. The result of this multiplication problem is $\\qquad$ .\n\n\n\nThe translation is as follows:\n\n6. Fill in the blanks with appropriate numbers to make the vertical calculation valid. The result of this multiplication problem is $\\qquad$ .",
        "answer": "726",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 117913,
        "difficulty": null
      },
      {
        "id": "78544110586fd72b3e2104a46ab6cc9ae2a409720c8f112e76ccbc0b1be8b7b3",
        "domain": "numina",
        "question": "# 4.1. Condition:\n\nIn front of the elevator stand people weighing 50, 51, 55, 57, 58, 59, 60, 63, 75, and 140 kg. The elevator's load capacity is 180 kg. What is the minimum number of trips needed to get everyone up?",
        "answer": "4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16492,
        "difficulty": null
      },
      {
        "id": "f5cf1ae465ecd685c42c6bc8056122f17e871003faa32745980817ec7251d7d1",
        "domain": "numina",
        "question": "12. Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=|2 x-1|+|3 x-2|+|4 x-3|+|5 x-4|$ is",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103283,
        "difficulty": null
      },
      {
        "id": "c48eb851080de82c20ba4398985902c1f6157dd6451248fb5128323678e631c0",
        "domain": "numina",
        "question": "7.1. Find the smallest natural number divisible by 5, with the sum of its digits being 100. Justify your answer.",
        "answer": "599999999995",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 22312,
        "difficulty": null
      },
      {
        "id": "21fe6cdff5952b5d1aeb2154daac8007c23064e30dc617ef4867fb6cccd68c96",
        "domain": "numina",
        "question": "7th Australian 1986 Problem A2 What is the smallest tower of 100s that exceeds a tower of 100 threes? In other words, let a 1 = 3, a 2 = 3 3 , and a n+1 = 3 a n . Similarly, b 1 = 100, b 2 = 100 100 etc. What is the smallest n for which b n > a 100 ?",
        "answer": "99",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 150465,
        "difficulty": null
      },
      {
        "id": "493ce07d4e82104e31d8584ddaba291bfc12a862f42e40c6f97fb896c7be23c6",
        "domain": "numina",
        "question": "13. Fox Alice and Cat Basilio (8th grade. 2 points). Every day, Cat Basilio and Fox Alice visit all 20 courtyards of the capital of the Country of Fools, and in each courtyard, they either receive or do not receive one gold coin with a probability of $\\frac{1}{2}$. If the number of gold coins collected by the end of the day is even, they split them equally. If it is odd, they split all but one coin equally, and the last coin is taken by Cat Basilio as the commercial director of the enterprise. Find the expected number of coins received by the cat per day.",
        "answer": "5.25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10906,
        "difficulty": null
      },
      {
        "id": "12012232b7d61940fafa0e201d5a16e2b620dcb1a332b40269c0fde4e7f55c07",
        "domain": "numina",
        "question": "2. In pentagon $A B C D E, A B=B C=C D=D E, \\angle B=96^{\\circ}$ and $\\angle C=\\angle D=108^{\\circ}$. Find $\\angle E$.\n(1 mark)\n在五邊形 $A B C D E$ 中, $A B=B C=C D=D E, \\angle B=96^{\\circ}$ 且 $\\angle C=\\angle D=108^{\\circ}$ 。\n求 $\\angle E$ 。",
        "answer": "102",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153582,
        "difficulty": null
      },
      {
        "id": "1a4b02da91ee6392fb7d6e24a4998186abf58a75daffd8a2d335c4375bfbc489",
        "domain": "numina",
        "question": "11. Let $A B C D E F$ be a convex equilateral hexagon such that lines $B C, A D$, and $E F$ are parallel. Let $H$ be the orthocenter of triangle $A B D$. If the smallest interior angle of the hexagon is 4 degrees, determine the smallest angle of the triangle $H A D$ in degrees.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151314,
        "difficulty": null
      },
      {
        "id": "9865947a0b75122566950aff3950616a63410ff75661684cccc5af7a45c01c30",
        "domain": "numina",
        "question": "19. Alice and Bob play a game in which two thousand and eleven $2011 \\times 2011$ grids are distributed between the two of them, 1 to Bob, and the other 2010 to Alice. They go behind closed doors and fill their grid(s) with the numbers $1,2, \\ldots, 2011^{2}$ so that the numbers across rows (left-to-right) and down columns (top-to-bottom) are strictly increasing. No two of Alice's grids may be filled identically. After the grids are filled, Bob is allowed to look at Alice's grids and then swap numbers on his own grid, two at a time, as long as the numbering remains legal (i.e. increasing across rows and down columns) after each swap. When he is done swapping, a grid of Alice's is selected at random. If there exist two integers in the same column of this grid that occur in the same row of Bob's grid, Bob wins. Otherwise, Alice wins. If Bob selects his initial grid optimally, what is the maximum number of swaps that Bob may need in order to guarantee victory?",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151322,
        "difficulty": null
      },
      {
        "id": "436e98c1c5477f11d70e2fbe32c4cea49ea96f7c06d2fe7a543c07473a51b7cc",
        "domain": "numina",
        "question": "11. Find the maximum and minimum values of the function $y=\\sqrt{x+27}+\\sqrt{13-x}+\\sqrt{x}$.\n\n untranslated portion:\n \n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。\n\n---\n\n11. Find the maximum and minimum values of the function $y=\\sqrt{x+27}+\\sqrt{13-x}+\\sqrt{x}$.",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92766,
        "difficulty": null
      },
      {
        "id": "d59ac5cdd2bab7872cb293db71aa1cd9ae65ca25ffd71b34cb96eb699a0e993b",
        "domain": "numina",
        "question": "## Task 1 - 340711\n\nArmin has 100 sticks of $7 \\mathrm{~cm}$ each and 100 sticks of $12 \\mathrm{~cm}$ each. He wants to lay out a line segment of $1 \\mathrm{~m}$ in length using such sticks. The sticks should always fit seamlessly together, and no part of the segment should be covered more than once.\n\nFind all the possibilities for how many $7 \\mathrm{~cm}$ sticks and how many $12 \\mathrm{~cm}$ sticks can be combined to form such a layout!",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 183875,
        "difficulty": null
      },
      {
        "id": "8112f028849f935c66b769ec90639be193670d58794641518b573114f4a24bc9",
        "domain": "numina",
        "question": "3. Through the point $(2,3)$ on the line $p: y=2 x-1$, a line $q$ perpendicular to the line $p$ was drawn. Find the area of the convex quadrilateral bounded by the lines $p, q$ and the coordinate axes.",
        "answer": "\\frac{19}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20579,
        "difficulty": null
      },
      {
        "id": "4a18a1323e7a8e1ad3d5edc051c9149d8426420d661db8cc9f5c80076f91b3e5",
        "domain": "numina",
        "question": "(14) From the left focus $F$ of the hyperbola $\\frac{x^{2}}{9}-\\frac{y^{2}}{16}=1$, draw a tangent to the circle $x^{2}+y^{2}=9$, with the point of tangency being $T$. Extend $F T$ to intersect the right branch of the hyperbola at point $P$. If $M$ is the midpoint of segment $F P$, and $O$ is the origin, find the value of $|M O|-| M T|$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 112578,
        "difficulty": null
      },
      {
        "id": "f44103fd984a9ed74a7e32652bbfc69b3c26f7da0dcdbe1ec923649922825d90",
        "domain": "numina",
        "question": "In the plane rectangular coordinate system $x O y$, the set of points\n$$\n\\begin{aligned}\nK= & \\{(x, y) \\mid(|x|+|3 y|-6) \\cdot \\\\\n& (|3 x|+|y|-6) \\leqslant 0\\}\n\\end{aligned}\n$$\n\ncorresponds to a plane region whose area is $\\qquad$",
        "answer": "24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98546,
        "difficulty": null
      },
      {
        "id": "c120c09c0e1ac5d5811f8696fcb11ef32e043eda0d36bc03c35661066dfa4552",
        "domain": "numina",
        "question": "## Task 2 - 080522\n\nIn a warehouse, there are three times as many kilograms of wheat as in a second one. After removing $85000 \\mathrm{~kg}$ from the first and $5000 \\mathrm{~kg}$ from the second, the stocks were equal.\n\nHow many tons of wheat were in the first and how many in the second warehouse before the removal?",
        "answer": "120",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 182963,
        "difficulty": null
      },
      {
        "id": "47391593ddb61160dd49457cd2c07fd641def1c46712ab149bf2e8d4728bf86a",
        "domain": "numina",
        "question": "1. Jeanne is standing in a single line with several other people. There are four people in front of her and seven people behind her. How many people are in the line?",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 115572,
        "difficulty": null
      },
      {
        "id": "8ba46f0a5f172423708067e8dad660d4eeed93d417fbfc2446f5c20acc99d8ff",
        "domain": "numina",
        "question": "11.5. For what minimum natural $n$ can $n$ distinct natural numbers $s_{1}, s_{2}, \\ldots, s_{n}$ be found such that $\\left(1-\\frac{1}{s_{1}}\\right)\\left(1-\\frac{1}{s_{2}}\\right) \\ldots\\left(1-\\frac{1}{s_{n}}\\right)=\\frac{7}{66} ?$",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9629,
        "difficulty": null
      },
      {
        "id": "5550d18bbd62fb1bf291441910e25bbf4cb5dee28228c6adc5e9861e8a5c75dd",
        "domain": "numina",
        "question": "## Zadatak A-1.1.\n\nOdredi $x_{1006}$ ako je\n\n$$\n\\begin{aligned}\n& \\frac{x_{1}}{x_{1}+1}= \\frac{x_{2}}{x_{2}+3}=\\frac{x_{3}}{x_{3}+5}=\\ldots=\\frac{x_{1006}}{x_{1006}+2011} \\\\\n& x_{1}+x_{2}+\\ldots+x_{1006}=503^{2}\n\\end{aligned}\n$$",
        "answer": "\\frac{2011}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88588,
        "difficulty": null
      },
      {
        "id": "a89a37a4e9395778b94456ae5974944de556d2124b69a13f80656aacd694b04f",
        "domain": "numina",
        "question": "12 Positive integer $n \\leqslant 500$, has the following property: For any element $m$ chosen from the set $\\{1,2, \\cdots, 500\\}$, the probability that $m$ divides $n$ is $\\frac{1}{100}$, then the maximum value of $n$ is $\\qquad$ .",
        "answer": "81",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 98067,
        "difficulty": null
      },
      {
        "id": "0b574d19558b984de279ae0c647b830b320aa7eff209b2270bd9ae457447d91c",
        "domain": "numina",
        "question": "1. On the board, three quadratic equations are written:\n\n$$\n\\begin{gathered}\n2020 x^{2}+b x+2021=0 \\\\\n2019 x^{2}+b x+2020=0 \\\\\nx^{2}+b x+2019=0\n\\end{gathered}\n$$\n\nFind the product of the roots of all the equations written on the board, given that each of them has two real roots.",
        "answer": "2021",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17447,
        "difficulty": null
      },
      {
        "id": "4e64b62e87c057a7b08dc551e12daabd664078ff45210ab26fd81d2e979c87a7",
        "domain": "numina",
        "question": "G3.1 Given that the solution of the equation $\\sqrt{3 x+1}+\\sqrt{3 x+6}=\\sqrt{4 x-2}+\\sqrt{4 x+3}$ is $a$, find the value of $a$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152948,
        "difficulty": null
      },
      {
        "id": "65fb95e281fa5a26d25ff5a984e956114d09e46910ce30cf081344539dac9187",
        "domain": "numina",
        "question": "7. Given the sequence $\\left\\{a_{n}\\right\\}$:\n$$\na_{n}=2^{n}+3^{n}+6^{n}+1\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nDoes there exist an integer $k \\geqslant 2$, such that $k$ is coprime with all numbers in the sequence $\\left\\{a_{n}\\right\\}$? If it exists, find the smallest integer $k$; if not, explain the reason. (Zhang Lei provided the problem)",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136823,
        "difficulty": null
      },
      {
        "id": "a599f31fd79d806abdbf7bfbd68309f3d67cf2e1f07c4248242c02ffe4a2bd58",
        "domain": "numina",
        "question": "Fedorov R.M.\n\nIn the Banana Republic, parliamentary elections were held in which all residents participated. All those who voted for the \"Mandarin\" party love mandarins. Among those who voted for other parties, 90% do not like mandarins. What percentage of the votes did the \"Mandarin\" party receive in the elections, if exactly $46\\%$ of the residents love mandarins?",
        "answer": "40",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 53178,
        "difficulty": null
      },
      {
        "id": "3a672e6146f0cf3aeb94c6a08bc334710d7d3fb88fa1a9396bb62cf873fb4702",
        "domain": "numina",
        "question": "1167. A locomotive is moving along a horizontal section of the track c) at a speed of 72 km/hour. In what time and at what distance will it be stopped by the brake, if the resistance to motion after the start of braking is equal to 0.2 of its weight.",
        "answer": "102",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 34390,
        "difficulty": null
      },
      {
        "id": "deef6a6de1ba7bd24a4b709e1b14685827a9329a9b6e006cd9845d378bcca2f4",
        "domain": "numina",
        "question": "## SUBJECT I\n\nSolve the equation $\\frac{x-2}{x}+\\frac{x-4}{x}+\\frac{x-6}{x}+\\ldots . . .+\\frac{2}{x}=12$",
        "answer": "50",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179556,
        "difficulty": null
      },
      {
        "id": "6d149d473d7d7c87ae82be1a75db272b5aee99f4df7d23e79fcb8983c4493b15",
        "domain": "numina",
        "question": "6. Let real numbers $a, b, c$ satisfy $a^{2}+b^{2}+c^{2}=1$, and let the maximum and minimum values of $a b+b c+c a$ be $M$ and $m$, respectively, then $M-m=$",
        "answer": "\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130567,
        "difficulty": null
      },
      {
        "id": "44e7df3f67703fb2075c8cde9f6f88fce0ee4fa2a248add3b8fdb1d9deb665b1",
        "domain": "numina",
        "question": "Coloring Vertex Degree\n\n20 phones are connected by wires in such a way that each wire connects two phones, each pair of phones is connected by no more than one wire, and no more than two wires extend from each phone. It is necessary to color the wires (each wire entirely in one color) so that the wires extending from each phone are of different colors. What is the smallest number of colors sufficient for such a coloring?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57695,
        "difficulty": null
      },
      {
        "id": "b931300a8ec23e23f7b89cffee02b9dc6e07239a7b8404ea0d6b33c20e2fc3ba",
        "domain": "numina",
        "question": "9.5. In the parliament of the island state of Promenade-and-Tornado, 2019 indigenous inhabitants were elected, who are divided into knights and liars: knights always tell the truth, liars always lie. At the first meeting, 2016 of them sat in the parliamentary seats arranged in the hall in a rectangle of $42 \\times 48$, three - in the seats of the chairman and his deputies in the presidium. During the meeting, each one stated that among their neighbors in the seat there are both knights and liars (neighbors are those who sit to the left, right, in front, behind, and diagonally: there can be from 3 to 8 in the hall and 1 or 2 in the presidium). Determine the minimum number of liars at the meeting.",
        "answer": "227",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18135,
        "difficulty": null
      },
      {
        "id": "03f317e654ea4854bb79487d448715166c511b28c612565753029791fa74fe5e",
        "domain": "numina",
        "question": "5. Given positive real numbers $a$ and $b$ satisfy $a+b=1$, then $M=$ $\\sqrt{1+a^{2}}+\\sqrt{1+2 b}$ the integer part is",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97730,
        "difficulty": null
      },
      {
        "id": "221b169cd33aca3bf11d10adb20ca2b178a0eadc578731e30acf53b3bedf606a",
        "domain": "numina",
        "question": "IMO 1991 Problem A3 Let S = {1, 2, 3, ... 280}. Find the smallest integer n such that each n-element subset of S contains five numbers which are pairwise relatively prime. Solution",
        "answer": "217",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 150333,
        "difficulty": null
      },
      {
        "id": "56c54057057d04efa1f2bd7ce7060a33c90351996cbeefc51947b2c5a007cbae",
        "domain": "numina",
        "question": "4. (1990 AIME Problem 8) \\( n \\) is the smallest positive integer that satisfies the following conditions:\n(1) \\( n \\) is a multiple of 75.\n(2) \\( n \\) has exactly 75 positive divisors (including 1 and itself).\nFind \\( \\frac{n}{75} \\).",
        "answer": "432",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 102003,
        "difficulty": null
      },
      {
        "id": "44757ebbd8cf247f3e75ff23e75d0481d813e7d4be61d00666841e0de2dd23f3",
        "domain": "numina",
        "question": "8. (10 points) In the inscribed quadrilateral $A B C D$, the degree measures of the angles are in the ratio $\\angle A: \\angle B: \\angle C=2: 3: 4$. Find the length of $A C$, if $C D=18, B C=12 \\sqrt{3}-9$.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19387,
        "difficulty": null
      },
      {
        "id": "d1193e44c0bb55bda30b0a177c6ac517e9ea6aecab7130808f51da1b70b742c7",
        "domain": "numina",
        "question": "Anička got a cake in the shape of a rectangle for her birthday. She cut it using two straight cuts. One cut went through the longer sides of the rectangle, intersecting them at one-third of their length, and the other cut went through the shorter sides, intersecting them at one-fifth of their length. Neither cut was parallel to the sides of the rectangle, and at each vertex of the rectangle, either two shorter segments or two longer segments of the divided sides always met.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_24585cf941083de07254g-3.jpg?height=402&width=552&top_left_y=553&top_left_x=752)\n\nAnička ate the gray-marked piece of the cake. Determine what fraction of the cake this was.\n\n(A. Bohiniková)",
        "answer": "\\frac{2}{15}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76745,
        "difficulty": null
      },
      {
        "id": "2f6cd9308e1f90baa8788c2f47ce69fc29981015135802461015c85a9324e3a2",
        "domain": "numina",
        "question": "G6.4 If $M=\\left(10^{2}+10 \\times 1+1^{2}\\right)\\left(10^{2}-1^{2}\\right)\\left(10^{2}-10 \\times 1+1^{2}\\right)$, find $M$.",
        "answer": "999999",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152577,
        "difficulty": null
      },
      {
        "id": "85d62b80f23cbdaefc7f7addcaa534233a2d81d939f690db13a73bcaab8afad8",
        "domain": "numina",
        "question": "10}\n\nPoints $A, B, C, D$ are marked on a sheet of paper. A recognition device can perform two types of operations with absolute accuracy: a) measure the distance in centimeters between two given points; b) compare two given numbers. What is the minimum number of operations this device needs to perform to definitely determine whether the quadrilateral $A B C D$ is a rectangle?",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58860,
        "difficulty": null
      },
      {
        "id": "fc3b833fee1ee270d29b1757838c082bdc82825039e6ac5c07a4d92ec8feb07f",
        "domain": "numina",
        "question": "10. Given positive real numbers $a, b$ satisfy: for real number $x$, when $1 \\leqslant x \\leqslant 4$, we have $a x+b-3 \\leqslant 0$. Then the minimum value of $\\frac{1}{a}-b$ is $\\qquad$",
        "answer": "1",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 117232,
        "difficulty": null
      },
      {
        "id": "7eee861d1f2866f5ccd1850f736054df12ad25bcc21f7c6a49b65e98f1e18123",
        "domain": "numina",
        "question": "20. Let $\\left\\{a_{n}\\right\\}$ and $\\left\\{b_{n}\\right\\}$ be sequences defined recursively by $a_{0}=2 ; b_{0}=2$, and $a_{n+1}=a_{n} \\sqrt{1+a_{n}^{2}+b_{n}^{2}}-b_{n}$; $b_{n+1}=b_{n} \\sqrt{1+a_{n}^{2}+b_{n}^{2}}+a_{n}$. Find the ternary (base 3) representation of $a_{4}$ and $b_{4}$.",
        "answer": "10000011001112222211100110000012",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151264,
        "difficulty": null
      },
      {
        "id": "fcb9c4ccd23e223dacf60740fbb4f7bfa0774e4820c05b5f4ef9c063249352e5",
        "domain": "numina",
        "question": "The lottery numbers drawn on February 28, 1969, were: 2, 34, 36, 50, 65. In connection with this, let's calculate the probability that in a lottery draw, the sum of the first two numbers, when arranged in ascending order, will be equal to the third number. What is the most frequent value of the third number in such draws?",
        "answer": "31",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 165183,
        "difficulty": null
      },
      {
        "id": "df4e966bbba796f7c51b0555bd242a0651d163f6b64ce419625e4c6c36eb556f",
        "domain": "numina",
        "question": "A1. A set $A$ is endowed with a binary operation $*$ satisfying the following four conditions:\n\n(1) If $a, b, c$ are elements of $A$, then $a *(b * c)=(a * b) * c$;\n\n(2) If $a, b, c$ are elements of $A$ such that $a * c=b * c$, then $a=b$;\n\n(3) There exists an element $e$ of $A$ such that $a * e=a$ for all $a$ in $A$; and\n\n(4) If $a$ and $b$ are distinct elements of $A \\backslash\\{e\\}$, then $a^{3} * b=b^{3} * a^{2}$, where $x^{k}=x * x^{k-1}$ for all integers $k \\geq 2$ and all $x$ in $A$.\n\nDetermine the largest cardinality $A$ may have.\n\n## BoJan Bašić, Serbia",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162365,
        "difficulty": null
      },
      {
        "id": "f4f36bbb469e49d68a6c5e276b0575f8e3b20509969bdbe367e3c7206b047e0c",
        "domain": "numina",
        "question": "\\section*{Exercise 3 - 241023}\n\nDetermine all natural numbers \\(z\\) for which the following holds:\n\nIf the last digit is removed from the digit representation of \\(z\\), the resulting digit representation is a divisor of \\(z\\).",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 185298,
        "difficulty": null
      },
      {
        "id": "c2ce219230e471254f52235ae6061436fca7a094df85aadb3c774b522ab505db",
        "domain": "numina",
        "question": "On an $A B=8 \\mathrm{~km}$ long road, buses travel in both directions at a speed of $12 \\mathrm{~km}$ per hour. The first bus from each location departs at 6 o'clock, and a new bus departs every 10 minutes.\n\nA pedestrian starts from $A$ towards $B$ at $81 / 4$ hours; their speed is $4 \\mathrm{~km}$ per hour.\n\nDetermine graphically how many oncoming buses the pedestrian encounters, and when and where.",
        "answer": "16",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 173192,
        "difficulty": null
      },
      {
        "id": "bb0d528ccfecf789f9ef5a85b0f99daa6e28954583814673994ee0de9013ea01",
        "domain": "numina",
        "question": "14th ASU 1980 Problem 12 Some unit squares in an infinite sheet of squared paper are colored red so that every 2 x 3 and 3 x 2 rectangle contains exactly two red squares. How many red squares are there in a 9 x 11 rectangle?",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 150652,
        "difficulty": null
      },
      {
        "id": "50db76b0a3cb15cf5c13ef15f756ac7e863a9ae41e3f64b2603ea62a4c09f66c",
        "domain": "numina",
        "question": "3. Petrov and Vasechkin were solving the same arithmetic problem. A certain number had to be divided by 2, multiplied by 7, and 1001 subtracted. Petrov performed all the operations correctly, while Vasechkin got everything mixed up: he divided by 8, squared the result, and also subtracted 1001. It is known that Petrov got a prime number. What number did Vasechkin get?",
        "answer": "295",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5850,
        "difficulty": null
      },
      {
        "id": "c369ccc4d001670002d3fc6350eb15d2322cfee1cacc6754fc2e4d1bb3c5f9b3",
        "domain": "numina",
        "question": "26. Given any 4-digit positive integer $x$ not ending in ' 0 ', we can reverse the digits to obtain another 4 -digit integer $y$. For example if $x$ is 1234 then $y$ is 4321 . How many possible 4-digit integers $x$ are there if $y-x=3177$ ?",
        "answer": "48",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160894,
        "difficulty": null
      },
      {
        "id": "8217681895488005d29ddb9da6b264d48ab104ffdd4b5834095833d22c0abd09",
        "domain": "numina",
        "question": "## SUBJECT 2\n\nFind the cardinality of the set:\n\n$$\nA=\\left\\{\\overline{a b c} / \\frac{a \\sqrt{2}}{\\sqrt{1}+\\sqrt{2}+\\sqrt{3}}-\\frac{b \\sqrt{3}}{\\sqrt{1}+\\sqrt{3}+\\sqrt{4}}-\\frac{2 c \\sqrt{8}}{\\sqrt{4}+\\sqrt{8}+\\sqrt{12}}=\\frac{-3}{\\sqrt{2}} ; a \\neq b \\neq c\\right\\}\n$$\n\nProf. Nicolae Jurubiță",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 180050,
        "difficulty": null
      },
      {
        "id": "41fcdeef5461fd940289d8444d83d8eb1019c114891fcf7ea0fdb44a6a0ed817",
        "domain": "numina",
        "question": "3B. Determine all natural numbers $x$ such that $x^{3}-x+p=507$, where $p$ is a prime number.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 190747,
        "difficulty": null
      },
      {
        "id": "458cb8beb640c95879f55c52fc3546e4ea155b6049c15dd40441b6915c3d4b43",
        "domain": "numina",
        "question": "Pedro wants to paint a cubic box in such a way that the faces that share an edge are painted in different colors. Calculate the minimum number of colors that will be needed to paint the box in this way.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74430,
        "difficulty": null
      },
      {
        "id": "06578f2ee827c1fd1a28205fbfbf12324b66a7daca480f30b14bb7b3179b5a4e",
        "domain": "numina",
        "question": "76. There are 13 different candies. In how many ways can a set be formed containing an odd number of candies",
        "answer": "2^{12}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44897,
        "difficulty": null
      },
      {
        "id": "44d1e8b4dcaed741f3743651fb0d71d547eae808749412d28823c8bcbe791258",
        "domain": "numina",
        "question": "6. Determine all possible values of the sum $a+b+c+d$, where $a, b, c, d$ are natural numbers satisfying the equation\n\n$$\n\\left(a^{2}-b^{2}\\right)\\left(c^{2}-d^{2}\\right)+\\left(b^{2}-d^{2}\\right)\\left(c^{2}-a^{2}\\right)=2021 .\n$$\n\n(Mária Dományová, Patrik Bak)",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 75906,
        "difficulty": null
      },
      {
        "id": "5afab7f66f49a2f95846d2b2300d7d084fb889f0d022c46204d776d7876f8602",
        "domain": "numina",
        "question": "Let $T$ be the set of all positive integer divisors of $2004^{100}$. What is the largest possible number of elements that a subset $S$ of $T$ can have if no element of $S$ is an integer multiple of any other element of $S$?",
        "answer": "10201",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 195125,
        "difficulty": null
      },
      {
        "id": "fbe4aec34f5263ecd959da648d881e067b7e2bf16a7bc2432539e84647c2aa27",
        "domain": "numina",
        "question": "## Task B-2.2.\n\nA natural number is said to be a palindrome if it reads the same from left to right and from right to left in decimal notation. We can order palindromes by size. Determine the 2010th palindrome in this order.",
        "answer": "1011101",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88038,
        "difficulty": null
      },
      {
        "id": "ed5205af8bbec99a51641c82901789f78668d2bb0843241532bd9c0b297a384b",
        "domain": "numina",
        "question": "37. We use one finger to represent a binary digit, so we can represent a binary number with our hands. A finger curled up represents the number 0, and a finger extended represents the number 1.\nFor example, in Figure 1, Guangtou Qiang's gesture represents the binary number $(10000)_2$, which is converted to a decimal number as follows: $1 \\times 16 + 0 \\times 8 + 0 \\times 4 + 0 \\times 2 + 0 \\times 1$, which equals 16.\nIn Figure 2, Guangtou Qiang's gesture represents the binary number $(1100)_2$, which is converted to a decimal number as follows: $1 \\times 8 + 1 \\times 4 + 0 \\times 2 + 0 \\times 1$, which equals 12.\n\nThen, in Figure 3, the gesture of Sailor Moon represents the binary number, which, when converted to a decimal number, equals $\\qquad$",
        "answer": "25",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 95382,
        "difficulty": null
      },
      {
        "id": "212362522f77019b4d41d0852e12bc0ab59637b421dd74ad1187639e60212ffe",
        "domain": "numina",
        "question": "3. For the parabola $y^{2}=4 a x(a>0)$, the focus is $A$. With $B(a+4,0)$ as the center and $|A B|$ as the radius, a semicircle is drawn above the $x$-axis, intersecting the parabola at two different points $M, N$. Let $P$ be the midpoint of the segment $M N$. (I) Find the value of $|A M|+|A N|$; (II) Does there exist such an $a$ that $|A M|,|A P|,|A N|$ form an arithmetic sequence?",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97937,
        "difficulty": null
      },
      {
        "id": "70d7baf5fbd377c416e832a5d57343424412fa2bddde57ba7da3eb913123d6d2",
        "domain": "numina",
        "question": "3. Find the number of natural numbers $k$, not exceeding 291000, such that $k^{2}-1$ is divisible by 291.",
        "answer": "4000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 2969,
        "difficulty": null
      },
      {
        "id": "1e888b9624d08b9d8bc88e5454bfbf4c12eca574a43f2de33a49e56c81a47e49",
        "domain": "numina",
        "question": "19.3. Can the number of diagonals of a polygon be exactly twice the number of its sides?\n\n$(6-8$ grades $)$",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 45037,
        "difficulty": null
      },
      {
        "id": "ed08a8ef340b23cd0b528a2ef982c8498264bac3e38a2e0b456328329cf35f91",
        "domain": "numina",
        "question": "Example 2.52. Calculate the area of the figure bounded by one arch of the cycloid $x=2(t-\\sin t), y=$ $=2(1-\\cos t)$.",
        "answer": "12\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30889,
        "difficulty": null
      },
      {
        "id": "d433192bf272a14cc0dd08d2fe0090d4f639aa1b48b0a4bee407c3b80c4e0f9b",
        "domain": "numina",
        "question": "15. [9] Let $a, b, c$ be positive integers such that\n$$\n\\frac{a}{77}+\\frac{b}{91}+\\frac{c}{143}=1 .\n$$\n\nWhat is the smallest possible value of $a+b+c$ ?",
        "answer": "79",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151809,
        "difficulty": null
      },
      {
        "id": "bb35ea1720d1227daaa33a30b679345ecadc41af262d5fea02fe41d30eb6ecde",
        "domain": "numina",
        "question": "Problem 4. Consider a $2018 \\times$ 2018 board. We call an LC tile a tile consisting of 9 congruent unit squares as shown in the figure below\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_fbc9a4a5e943c67ef798g-3.jpg?height=143&width=157&top_left_y=2341&top_left_x=961)\n\nWhat is the maximum number of LC tiles that can be placed without overlapping on the board? (Each of the 9 unit squares of the tile must overlap with one of the unit squares of the board; a tile can be rotated, mirrored, etc.)\n\nAlexandru Gîrban",
        "answer": "672^2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 179820,
        "difficulty": null
      },
      {
        "id": "069b25bb1c76a7f93d41652b57b66b30598320e008abe63ff60adeb1efb88f0e",
        "domain": "numina",
        "question": "$10 \\cdot 10$ Given a four-digit number that satisfies the following conditions:\n(1) If the units digit and the hundreds digit, as well as the tens digit and the thousands digit, are simultaneously swapped, then its value increases by 5940;\n(2) When divided by 9, the remainder is 8.\nFind the smallest odd number among these four-digit numbers.\n(Shandong Province, China Mathematics Competition, 1979)",
        "answer": "1979",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 110123,
        "difficulty": null
      },
      {
        "id": "db6edf8d7250b374bd06eecf2ca78807e01b9ca5c18b9fa9f5c2e037d93db008",
        "domain": "numina",
        "question": "8-1. Two rectangles $8 \\times 10$ and $12 \\times 9$ are overlaid as shown in the figure. The area of the black part is 37. What is the area of the gray part? If necessary, round the answer to 0.01 or write the answer as a common fraction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1d29d5cb430346a58eb5g-15.jpg?height=471&width=577&top_left_y=684&top_left_x=745)",
        "answer": "65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12368,
        "difficulty": null
      },
      {
        "id": "0d511acdd529546516f96f1c872694c14cdd5ce42ad8ccf29ad55438eaab438f",
        "domain": "numina",
        "question": "1308. Calculate the volume of the body obtained by rotating around the $O x$ axis the figure bounded by the parabolas $y=x^{2}$ and $x=y^{2}$.",
        "answer": "\\frac{3}{10}\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30598,
        "difficulty": null
      },
      {
        "id": "7315f0cf9010a4a9915c34eb53adf7f949029b2c146e8a103aa0f3d745945cbd",
        "domain": "numina",
        "question": "We wrote the numbers from 1 to 2009 on a piece of paper. In the second step, we also wrote down twice each of these numbers on the paper, then erased the numbers that appeared twice. We repeat this step in such a way that in the $i$-th step, we also write down $i$ times each of the numbers $1, 2, \\ldots, 2009$ on the paper, then erase the numbers that appear twice. How many numbers will be on the paper after the 2009th step?",
        "answer": "2009",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169773,
        "difficulty": null
      },
      {
        "id": "152fa4db5751434df0795c8dd8aba2d2ab99b5ca2fe58f19485ee4a79091de91",
        "domain": "numina",
        "question": "8.2. Points $A, B, C, D$ are the centers of the circles inscribed in the faces $P Q S, P R S$, $Q R S$ and $P Q R$ of the tetrahedron $P Q R S$. It is known that $P Q=R S=7, P R=Q S=8, P S=Q R=9$. Find the volume of the tetrahedron $A B C D$. Round your answer to the nearest hundredth if necessary.",
        "answer": "1.84",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6269,
        "difficulty": null
      },
      {
        "id": "0fe8e463100e440e50fa54f8912c2e3badb503d49745f76ff34e805d53d4e43f",
        "domain": "numina",
        "question": "8.5. On an $8 \\times 8$ chessboard, $k$ rooks and $k$ knights are placed such that no figure attacks any other. What is the largest $k$ for which this is possible?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 15754,
        "difficulty": null
      },
      {
        "id": "d79d8f75032fc1486feba55ce55eb4da777c53632315a8058371e7da438385e8",
        "domain": "numina",
        "question": "10. In the methods for finding the zeros of a function, there is one called the Newton's method. The process of this method is: for the function $f(x)$, given $x_{1}$, and the intersection of the tangent line of $f(x)$ passing through the point $Q_{n}\\left(x_{n}, f\\left(x_{n}\\right)\\right)$ with the $x$-axis is denoted as $x_{n+1}(n=1,2,3, \\ldots \\ldots)$, when $n$ is sufficiently large, we can use $x_{n}$ as an approximate value of the zero. Let $f(x)=x^{2}-2, x_{1}=2$, find the recursive formula of the sequence $\\left\\{x_{n}\\right\\}$, and find its limit.",
        "answer": "\\sqrt{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 142488,
        "difficulty": null
      },
      {
        "id": "c6b4a3fab47b4c1f4bfdbc38e1aadd921ca1451eff6fe16aa86ca70619187e4b",
        "domain": "numina",
        "question": "What is the maximum number of kings that can be placed on an $8 \\times 8$ chessboard without any of them attacking each other?\n\n#",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 54108,
        "difficulty": null
      },
      {
        "id": "5aae7c5f660a84df2ae8b6c57025c2e533944b80b17efd53776d675d209ead89",
        "domain": "numina",
        "question": "Example 2 Let $S=\\{1,2,3,4\\}, n$ terms of the sequence: $a_{1}, a_{2}, \\cdots, a_{n}$ have the following property: for any non-empty subset $B$ of $S$ (the number of elements in $B$ is denoted by। $B$ ।), there are adjacent। $B$ ।terms in the sequence that exactly form the set $B$. Find the minimum value of $n$.\n(1997 Shanghai Competition Question)",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104100,
        "difficulty": null
      },
      {
        "id": "37c0035d84edd17f886bc184c7ca587ed5f3087b7b94c49632443e2426fa793c",
        "domain": "numina",
        "question": "39. Junior high school student Xiao Peng was surprised to find that: his age this year (2018) is exactly the sum of the four digits of his birth year plus 10. He told this discovery to his math teacher, Mr. Wang, who thought for a moment and said: My age this year is also exactly the sum of the four digits of my birth year plus 10. So, Mr. Wang is $\\qquad$ years older than Xiao Peng.",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114807,
        "difficulty": null
      },
      {
        "id": "a1e40649d62127ca0fa318c9d3e47889b0b37f6254960abc2056d69a68418e4a",
        "domain": "numina",
        "question": "10. Washing a pile of vegetables to remove residual pesticides with water. The effect of washing once with a certain amount of water is assumed as follows: 1 unit of water can remove $\\frac{1}{2}$ of the residual pesticides on the vegetables. The more water used, the more pesticides are removed, but there will always be some pesticides left on the vegetables. Let the ratio of the residual pesticide amount on the vegetables after washing once with $x$ units of water to the residual pesticide amount before washing be the function $f(x)$.\n(1) Specify the value of $f(0)$ and explain its practical significance;\n(2) Write down the conditions and properties that the function $f(x)$ should satisfy based on the assumptions;\n(3) Suppose $f(x)=\\frac{1}{1+x^{2}}$. Given $a (a>0)$ units of water, you can wash once, or you can divide the water into 2 equal parts and wash twice. Which method leaves fewer residual pesticides on the vegetables? Explain your reasoning.",
        "answer": "2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123314,
        "difficulty": null
      },
      {
        "id": "e28436132fd849337c9900f3de63c15b89c7d6e3c42ae2fb9d572fee3ac3e95c",
        "domain": "numina",
        "question": "(13) As shown in the figure, in the Cartesian coordinate system, the circle $M$ with the equation $x^{2}+y^{2}+D x+E y+F=0$ has an inscribed quadrilateral $A B C D$ whose diagonals $A C$ and $B D$ are perpendicular to each other, and $A C$ and $B D$ lie on the $x$-axis and $y$-axis, respectively.\n(1) Prove that $F<0$;\n(2) If the area of quadrilateral $A B C D$ is 8, the length of diagonal $A C$ is 2, and $\\overrightarrow{A B} \\cdot \\overrightarrow{A D}=0$, find the value of $D^{2}+E^{2}-4 F$.",
        "answer": "64",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 95591,
        "difficulty": null
      },
      {
        "id": "ef2767e8af541f50c2055b526616d46080ef916dbe4182e8a0816d12d49d8fd0",
        "domain": "numina",
        "question": "12. Restore the digits. On the board, there was the product of three consecutive even numbers. During the break, Vasya erased some digits. As a result, on the board remained $87 * * * * * 8$. Help Petya find the missing digits in the product.",
        "answer": "87526608",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35621,
        "difficulty": null
      },
      {
        "id": "f58a046ac815198ddd738f785c9aef5139a802d1abe6e7c124c9af10800baad3",
        "domain": "numina",
        "question": "5. The minimum value of the function $y=\\sqrt{x^{2}-2 x+5}+\\sqrt{x^{2}-4 x+13}$ is",
        "answer": "\\sqrt{26}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102612,
        "difficulty": null
      },
      {
        "id": "dbf709762f0ad5dc5d4d593c5ad60cf2221a13959c3a13126d4a8af21fee03a6",
        "domain": "numina",
        "question": "## Task A-4.1.\n\nFor a natural number $n$, let $s(n)$ denote the sum of its positive divisors, and $d(n)$ the number of its positive divisors. Determine all natural numbers $n$ such that\n\n$$\ns(n)=n+d(n)+1\n$$",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88429,
        "difficulty": null
      },
      {
        "id": "cdab796a41bddfa97de5365b044587fdc750fe8a77198fc5ede6b5da8b0e6c89",
        "domain": "numina",
        "question": "7. Let $S=\\left\\{(x, y) \\in \\mathbb{Z}^{2} \\mid 0 \\leq x \\leq 11,0 \\leq y \\leq 9\\right\\}$. Compute the number of sequences $\\left(s_{0}, s_{1}, \\ldots, s_{n}\\right)$ of elements in $S$ (for any positive integer $n \\geq 2$ ) that satisfy the following conditions:\n- $s_{0}=(0,0)$ and $s_{1}=(1,0)$,\n- $s_{0}, s_{1}, \\ldots, s_{n}$ are distinct,\n- for all integers $2 \\leq i \\leq n, s_{i}$ is obtained by rotating $s_{i-2}$ about $s_{i-1}$ by either $90^{\\circ}$ or $180^{\\circ}$ in the clockwise direction.",
        "answer": "646634",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151924,
        "difficulty": null
      },
      {
        "id": "7ba40f8ed1bce0d0063b61960ac894b7313e8494890453fea2b33dd01ba6d206",
        "domain": "numina",
        "question": "19. Next year's 'leap day', 29th February 2024 , may be written in 'ddmmyy' format as the number 290224. What is the sum of all the two-digit factors of 290224 ?",
        "answer": "397",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156369,
        "difficulty": null
      },
      {
        "id": "2da6a6eacf4564ea3097685a5de732fe290f76e0d7ff3b1e0f30903096e55eab",
        "domain": "numina",
        "question": "3. Three brigades, working together, must complete a certain job. It is known that the first and second brigades together can complete it 36 minutes faster than the third brigade. In the time it takes for the first and third brigades to complete the job together, the second brigade can complete only half of the job. In the time it takes for the second and third brigades to complete the job together, the first brigade can complete $\\frac{2}{7}$ of the job. How long will it take for all three brigades to complete the job together?",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14721,
        "difficulty": null
      },
      {
        "id": "7c504f5dfa9e555290e8d4359abeda93b25e601c2b458e11826dca5afd451cc7",
        "domain": "numina",
        "question": "9.1. For non-negative numbers $a, b, c, d$, the following equalities are satisfied:\n\n$\\sqrt{a+b}+\\sqrt{c+d}=\\sqrt{a+c}+\\sqrt{b+d}=\\sqrt{a+d}+\\sqrt{b+c}$. What is the maximum number of distinct values that can be among the numbers $a, b, c, d$?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9823,
        "difficulty": null
      },
      {
        "id": "87c81c37b916feafa325c96e102fde924766aa6482c79a107137b4cfc930c463",
        "domain": "numina",
        "question": "4. Find the smallest four-digit number that is equal to the square of the sum of the numbers formed by its first two digits and its last two digits.",
        "answer": "2025",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 34791,
        "difficulty": null
      },
      {
        "id": "90bef54c412abdc925783905d07e55085621ecf8ca2e00705d641cd31cef902b",
        "domain": "numina",
        "question": "14. Garden plot. The length of a rectangular plot of land was increased by $35 \\%$, and the width was decreased by $14 \\%$. By what percentage did the area of the plot change?",
        "answer": "16.1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 35623,
        "difficulty": null
      },
      {
        "id": "a3d5a7078297b0b1d0b7877c8e8bdbfcb74abf15eb1dd7c05d016680e6667524",
        "domain": "numina",
        "question": "1. If we insert a zero between the digits of a two-digit number, we get a number that is nine times larger. Write down all such two-digit numbers.",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 163714,
        "difficulty": null
      },
      {
        "id": "be2eeab2243b3632fd5936982a8e833587cdb085bd95046d715152fa15e2b63d",
        "domain": "numina",
        "question": "Problem 10. The sum of the surface areas of the polyhedra into which a parallelepiped is divided by sections is equal to the sum of the surface area of the parallelepiped and the areas of the internal surfaces. The sum of the areas of the internal surfaces is equal to twice the sum of the areas of the sections.\n\nLet's find the maximum possible area of a section passing through the diagonal $XY$ of an arbitrary parallelepiped with edges $a \\leq b \\leq c$. The section is a parallelogram $ZXTY$, whose vertices lie on opposite edges of the parallelepiped. The area of the parallelogram is the product of the length of the diagonal $XY$ and the distance from point $Z$ to $XY$.\n\nConsider the projection of the parallelepiped onto a plane perpendicular to the diagonal $XY$. From the diagram, it is clear that the distance from point $Z$ to the broken line $ABC$, and thus to the diagonal $XY$, is greatest if\n\n$Z$ coincides with one of the vertices $A, B$, or $C$.\n\nThis means that the section passes through one of the edges of the parallelepiped. Therefore, the section with the largest area is one of the diagonal sections. All these sections are rectangles. Let's find the largest of their areas:\n\n$$\nS_{1}=a \\sqrt{b^{2}+c^{2}}, \\quad S_{2}=b \\sqrt{a^{2}+c^{2}}, \\quad \\text{and} \\quad S_{3}=c \\sqrt{b^{2}+a^{2}}\n$$\n\nFrom the condition $a \\leq b \\leq c$, it follows that\n\n$a^{2} b^{2}+a^{2} c^{2} \\leq c^{2} b^{2}+a^{2} c^{2}$, and $a^{2} b^{2}+c^{2} b^{2} \\leq c^{2} b^{2}+a^{2} c^{2}$. Therefore, $S_{1} \\leq S_{3}$ and $S_{2} \\leq S_{3}$. This means that the section with the largest area passes through the largest edge. According to the condition, the largest length is the edge $AA_{1}$, so the sections $AA_{1}C_{1}C$ and $BB_{1}D_{1}D$ have the largest area:\n\n$5 \\sqrt{4^{2}+3^{2}}=25$. The sum of the surface areas of the polyhedra into which the parallelepiped is divided by these sections (see the diagram) is\n\n$$\n2\\left(A A_{1} \\cdot A B + A A_{1} \\cdot A D + A B \\cdot A D\\right) + 4 \\cdot 25 = 194\n$$",
        "answer": "194",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9345,
        "difficulty": null
      },
      {
        "id": "e42a8eab945dad132a4361628d4067631c30d74db390a7f0cb051d974f98f6cd",
        "domain": "numina",
        "question": "\\section*{Task 1 - 011021}\n\nIn 1970, the Soviet Union was to produce at least 900 billion kWh and in 1980 at least 2700 billion kWh of electricity. For the USA, the Federal Energy Commission assumes 1475 billion kWh and 2230 billion kWh, respectively.\n\nWhen would the USSR have overtaken the USA in electricity production if a constant increase in energy production is assumed?",
        "answer": "1976",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184854,
        "difficulty": null
      },
      {
        "id": "3cb96ec17dbb78415ac2fe0f50a74509ba2a613c147fe50439229a85d1425232",
        "domain": "numina",
        "question": "7-3. A secret object is a rectangle measuring $200 \\times 300$ meters. Outside the object, there is a guard at each of the four corners. An intruder approached the perimeter of the secret object from the outside, and all the guards ran to him along the shortest paths along the external perimeter (the intruder remained in place). Three of the guards ran a total of 850 meters to reach the intruder. How many meters did the fourth guard run to reach the intruder?",
        "answer": "150",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14079,
        "difficulty": null
      },
      {
        "id": "8a295d7ed266e1217e464d997761eb1bd0f588f09564c1cf520c76389e75650a",
        "domain": "numina",
        "question": "1. Determine the number of pairs $(a, b)$ of natural numbers $(1 \\leqq a<b \\leqq 86)$ for which the product $a b$ is divisible by three.",
        "answer": "2002",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75505,
        "difficulty": null
      },
      {
        "id": "4b8705550333bfe6060f5691e67bed99ee0a2c5ca8c8682fc735c224619593aa",
        "domain": "numina",
        "question": "1. (8 points) The sum of the digits of the result of the expression $999999999-88888888+7777777-666666+55555-4444+333-22+1$ is $\\qquad$ .",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 120724,
        "difficulty": null
      },
      {
        "id": "1646820d6e220ad0452dddaf80b12da98a02fcf57189fd2373e7f58ee38c1393",
        "domain": "numina",
        "question": "## 233. Math Puzzle 10/84\n\nMatthias rides a 3 km long street at a constant speed of $v=15 \\mathrm{~km} / \\mathrm{h}$.\n\nHow often and how many city buses overtake him, which run at 5-minute intervals with $v=40 \\mathrm{~km} / \\mathrm{h}$?",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181904,
        "difficulty": null
      },
      {
        "id": "c70d80428a0dbe604dd65fe2a97838141725ee65232b336000c324f266b7bf9c",
        "domain": "numina",
        "question": "10. (6 points) Count the number of squares with \"mei\" in the figure: $\\qquad$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 115844,
        "difficulty": null
      },
      {
        "id": "19c4f81868a42828e46b3ea8c245af1f2928397cb7368ca4c03adcf677a1b761",
        "domain": "numina",
        "question": "10. $A B C D-A_{1} B_{1} C_{1} D_{1}$ is a cube with edge length 1, $D 、 E 、 F 、 G$ are the midpoints of $B A$ 、 $B C 、 B B_{1}$ and $A_{1} D_{1}$, respectively, then the volume of the tetrahedron $G-D E F$ is",
        "answer": "\\frac{1}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 131352,
        "difficulty": null
      },
      {
        "id": "f8d5d2acb65827605ed69e74af9863227b4948c27501ccae01f876a7dfb0cba9",
        "domain": "numina",
        "question": "21. In the diagram below, $A B C D$ is a convex quadrilateral such that $A C$ intersects $B D$ at the midpoint $E$ of $B D$. The point $H$ is the foot of the perpendicular from $A$ onto $D E$, and $H$ lies in the interior of the segment $D E$. Suppose $\\angle B C A=90^{\\circ}, C E=12 \\mathrm{~cm}, E H=15 \\mathrm{~cm}$, $A H=40 \\mathrm{~cm}$ and $C D=x \\mathrm{~cm}$. Find the value of $x$.\n21. Answer: 40\nLet $M$ be the midpoint of $A H$ and $N$ the midpoint of $B C$. Then $C D$ is parallel to $N E$ and $C D=2 N E$. Since $\\triangle A H E$ is similar to $\\triangle B C E$, we have $\\triangle M H E$ is similar to $\\triangle N C E$. As $M E=\\sqrt{15^{2}+20^{2}}=25$, we thus have $N E=\\frac{E O}{E H} \\cdot M E=\\frac{12 \\times 25}{15}=20$. Therefore $C D=2 N E=40$.",
        "answer": "40",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160713,
        "difficulty": null
      },
      {
        "id": "4b338283fd1951afd6132a1330742374993b217cb25169efe2a2392288eac263",
        "domain": "numina",
        "question": "5. In $\\triangle A B C$, $\\angle C=90^{\\circ}, \\angle B=30^{\\circ}$, $A C=1$, $M$ is the midpoint of $A B$. Fold $\\triangle A C M$ along $C M$ so that the distance between points $A$ and $B$ is $\\sqrt{2}$. Then the distance from point $A$ to the plane $B C M$ is $\\qquad$ .",
        "answer": "\\frac{\\sqrt{6}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113405,
        "difficulty": null
      },
      {
        "id": "94b6e2cc9a9566cddf216bab812dda2f74f7e474ded021ca680e05d1c8b4dbc5",
        "domain": "numina",
        "question": "1. Circle $k_{1}$ with radius 1 has an external tangency $\\mathrm{s}$ with circle $k_{2}$ with radius 2. Each of the circles $k_{1}, k_{2}$ has an internal tangency with circle $k_{3}$ with radius 3. Calculate the radius of circle $k$, which has external tangency $\\mathrm{s}$ with circles $k_{1}, k_{2}$ and internal tangency $\\mathrm{s}$ with circle $k_{3}$.",
        "answer": "\\frac{6}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75888,
        "difficulty": null
      },
      {
        "id": "9cc635596159c320ab65b2e2d775506f9424eb145efd34fc339048268cd5d88d",
        "domain": "numina",
        "question": "8,9 |\n\nFind the cosine of the angle at the base of an isosceles triangle if the point of intersection of its altitudes lies on the inscribed circle of the triangle.\n\n#",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 54300,
        "difficulty": null
      },
      {
        "id": "f21da74005821cf875e990d9ebbdbce51b92e481124c52b442e6222c0acf9327",
        "domain": "numina",
        "question": "19. Given a quadrilateral pyramid $P-ABCD$, the base $ABCD$ is a rhombus with side length 2, $\\angle ABC=60^{\\circ}$, $E$ is the midpoint of $AB$, $PA \\perp$ plane $ABCD$, and the sine of the angle between $PC$ and plane $PAD$ is $\\frac{\\sqrt{6}}{4}$.\n(1) Find a point $F$ on edge $PD$ such that $AF \\parallel$ plane $PEC$;\n(2) Find the cosine of the dihedral angle $D-PE-A$.",
        "answer": "\\frac{4\\sqrt{31}}{31}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 138011,
        "difficulty": null
      },
      {
        "id": "ed64fe84086b9913e40f2d0bf682479965e6bc7b6d3d35804d90441bac429644",
        "domain": "numina",
        "question": "10. In the packaging workshop, there are 2 types of boxes: one for 20 parts and one for 27 parts. A batch of parts between 500 and 600 pieces has arrived for packaging. When the parts are packed in the first type of box, 13 parts are left unpacked, and when packed in the second type of box, 7 spaces are left unfilled. How many parts were in the batch?",
        "answer": "533",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19161,
        "difficulty": null
      },
      {
        "id": "71132f600f0c1f7d7823dee2f4b9d7263f8e1cc1f6a64a6836e254a797b3c578",
        "domain": "numina",
        "question": "6. In $\\triangle A B C$, $a, b, c$ are the sides opposite to the internal angles $A, B, C$ respectively. If $7 b^{2}+25 c^{2}-25 a^{2}=0$. Try to find the value of $\\sin (2 A-C)$ when $A-C$ reaches its maximum value.",
        "answer": "\\frac{117}{125}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 96616,
        "difficulty": null
      },
      {
        "id": "d773450fe81128518c1c01ab4e476663ba9b3ce57a1f335050464fdfebdf46ec",
        "domain": "numina",
        "question": "6. Let the result of a single coin toss be denoted as $A$ if it lands heads up; as $B$ if it lands tails up. When tossing the coin several times in a row, we can obtain a sequence composed of $A$ and $B$, for example, tossing six times might result in the sequence $A A B A B B$. In the resulting sequence, we can count the occurrences of patterns like $A B$. For instance, in $A A B A B B$, $A B$ appears twice. When tossing the coin 16 times in a row, among all possible sequences, there are $\\qquad$ sequences that satisfy: $A A$ appears 2 times, $B B$ appears 6 times, $A B$ appears 4 times, and $B A$ appears 4 times.",
        "answer": "840",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117428,
        "difficulty": null
      },
      {
        "id": "ec006c02ecdd2f2872b03b6beaacd122d4058f5ed15c74d5b66e944e38445008",
        "domain": "numina",
        "question": "10.4. A chord is at a distance $h$ from the center of a circle. In each of the two segments of the circle subtended by this chord, a square is inscribed such that a pair of its adjacent vertices lies on the chord, and the other pair of adjacent vertices lies on the corresponding arc of the circle. Find the difference in the lengths of the sides of the squares.",
        "answer": "\\frac{8}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 18541,
        "difficulty": null
      },
      {
        "id": "cfedb527423ad07b1f99244f5ce2ffb9abc24f4954117bb00cfb7ed63d955194",
        "domain": "numina",
        "question": "Exercise 3. In the figure below, \\( AB = AF = CD = DE = 18 \\), \\( BC = EF = a \\), and \\( BF = 24 \\). Additionally, the quadrilateral \\( BCEF \\) is a rectangle.\n\n![](https://cdn.mathpix.com/cropped/2024_06_04_d69c62b301e4ba44b0f0g-2.jpg?height=238&width=512&top_left_y=1949&top_left_x=796)\n\nWe assume that the area of the hexagon \\( ABCDEF \\) is equal to the area of a rectangle whose two consecutive sides have lengths \\( a \\) and 36. Find \\( a^2 \\). Justify your answer.\n\nNote. You may use the Pythagorean theorem, which states that if \\( XYZ \\) is a right triangle at \\( X \\), then \\( XY^2 + XZ^2 = YZ^2 \\).",
        "answer": "720",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 78208,
        "difficulty": null
      },
      {
        "id": "4d1f5c243ce68cfcabfc702950def686783d3232ccc6486ceda81eaaee8024b6",
        "domain": "numina",
        "question": "## Task 2\n\nAs the special train stops, a freight train passes by. Peter counts the cars: Right after the locomotive, 5 closed cars pass. Then follow 4 times as many open cars. At the end, 6 tank cars pass.\n\nHow many cars does the freight train have?",
        "answer": "31",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186465,
        "difficulty": null
      },
      {
        "id": "14e45f5ae146d8ff4131897d62a321340657a9bd353af758df72d0d2dbd1353c",
        "domain": "numina",
        "question": "## Task 1\n\nCalculate the sum of the numbers 6 and 4. Add the number 30 to this sum.",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186505,
        "difficulty": null
      },
      {
        "id": "5ed084737d90e9a36cd29a06ad2e437f21466a91062b64ccf3ffc3a738a2a8e5",
        "domain": "numina",
        "question": "18. Let $m$ and $n$ be relatively prime positive integers. If $m^{3} n^{5}$ has 209 positive divisors, then how many positive divisors does $m^{5} n^{3}$ have?",
        "answer": "217",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154749,
        "difficulty": null
      },
      {
        "id": "a560be633994aa814cdf1cf4622dacbe749636cef1b6d6a2cf775dc6c2cc9372",
        "domain": "numina",
        "question": "13. Given three different prime numbers $p$, $q$, and $r$ that satisfy $p q r = 18 \\underbrace{99 \\ldots 9}_{n \\uparrow 9} 62$, define $f(n)$ as the sum of the digits of the natural number $n$. Then $f(p) + f(q) + f(r) - f(p q r) =$ $\\qquad$",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 131612,
        "difficulty": null
      },
      {
        "id": "bc5745c8816833ac23ffc769d5210e218a7423f9314f2bfbbd9b238db3dfcce4",
        "domain": "numina",
        "question": "13. When the value of the natural number $n$ is taken as $1,2,3, \\cdots, 2015$, the expression $\\left[\\frac{n}{2}\\right]+\\left[\\frac{n}{3}\\right]+\\left[\\frac{n}{5}\\right]$ has $\\qquad$ different values. (Note: $[x]$ represents the largest natural number not exceeding $x$)",
        "answer": "1479",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 141620,
        "difficulty": null
      },
      {
        "id": "866cc3b2d5afba5f29206bd81478ac623dccc9718398ead0624fd5d62e42ac6d",
        "domain": "numina",
        "question": "## Task 4 - 130614\n\nJörg and Claudia are arguing about whether there are more natural numbers from 0 to 1000 whose decimal representation contains (at least) one 5, or whether there are more that do not contain a 5.\n\nDetermine the correct answer to this question!",
        "answer": "271",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183266,
        "difficulty": null
      },
      {
        "id": "e3aa3330e79b5c95992a261ac54497a12372e1d56421982ca8fa07717cb38600",
        "domain": "numina",
        "question": "Task B-1.3. An amount of 18200 kuna needs to be divided among three people so that each subsequent person receives $20 \\%$ more than the previous one. How much money will each of them receive?",
        "answer": "5000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87910,
        "difficulty": null
      },
      {
        "id": "286d78c80bca107d883a7aedc3a07942100b05fc9cf97bcedabb36c2e78f3480",
        "domain": "numina",
        "question": "2. To walk 4 km, ride 6 km on a bicycle, and drive 40 km by car, Uncle Vanya needs 2 hours and 12 minutes. If he needs to walk 5 km, ride 8 km on a bicycle, and drive 30 km by car, it will take him 2 hours and 24 minutes. How much time will Uncle Vanya need to walk 8 km, ride 10 km on a bicycle, and drive 160 km by car?",
        "answer": "5.8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8553,
        "difficulty": null
      },
      {
        "id": "65b8a037d519d229fa120994ad242ab9a928c68df7873c902d1fed0ce9d1a100",
        "domain": "numina",
        "question": "Diding M.\nThere are 100 piles, each with 400 stones. In one move, Petya selects two piles, removes one stone from each, and earns as many points as the absolute difference in the number of stones in these two piles. Petya must remove all the stones. What is the maximum total number of points he can earn?",
        "answer": "3920000",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56376,
        "difficulty": null
      },
      {
        "id": "feaf2fd9942e9b540b25564e45c0ad490d2bd15d9c76baeb922a298f24a44e8a",
        "domain": "numina",
        "question": "8. (3 points) Mr. Wang, who teaches mathematics, is preparing to visit a friend. Before leaving, Mr. Wang calls his friend, whose phone number is 27433619. After Mr. Wang finishes the call, he realizes that this phone number is exactly the product of 4 consecutive prime numbers. What is the sum of these 4 prime numbers? $\\qquad$",
        "answer": "290",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 108709,
        "difficulty": null
      },
      {
        "id": "580f3768b08425fb19bd0b31a90c95c285090a2ac6d69591289ae6867fc7d10f",
        "domain": "numina",
        "question": "12. C5 (NZL) Let $N$ be a positive integer. Two players $A$ and $B$, taking turns, write numbers from the set $\\{1, \\ldots, N\\}$ on a blackboard. $A$ begins the game by writing 1 on his first move. Then, if a player has written $n$ on a certain move, his adversary is allowed to write $n+1$ or $2 n$ (provided the number he writes does not exceed $N$ ). The player who writes $N$ wins. We say that $N$ is of type $A$ or of type $B$ according as $A$ or $B$ has a winning strategy.\n(a) Determine whether $N=2004$ is of type $A$ or of type $B$.\n(b) Find the least $N>2004$ whose type is different from that of 2004 .",
        "answer": "2048",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 158865,
        "difficulty": null
      },
      {
        "id": "b91dd3deb93e24a666b3a5998e72e079769e5885d4ad8b4d9cca08e6e750a595",
        "domain": "numina",
        "question": "2. (15 points) Let $f(x)=x^{2}-5 x+2020$. Solve the equation $f(3-x)=f(3 x-1)$",
        "answer": "1.51",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10130,
        "difficulty": null
      },
      {
        "id": "4bdd0ac61e48699f76885e75d0ce15419840a169b35242a3f292c5a567d21c4c",
        "domain": "numina",
        "question": "2. It is known that the numbers $x, y, z$ form an arithmetic progression in the given order with a common difference $\\alpha=\\arccos \\frac{2}{3}$, and the numbers $\\frac{1}{\\sin x}, \\frac{6}{\\sin y}, \\frac{1}{\\sin z}$ also form an arithmetic progression in the given order. Find $\\sin ^{2} y$.",
        "answer": "\\frac{5}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3304,
        "difficulty": null
      },
      {
        "id": "bba82a028a91e676f0c7abe6c41aa266f50b15e019d81ddc3040a5999cfbd5bf",
        "domain": "numina",
        "question": "4.031. It is known that the interior angles of a certain convex polygon, the smallest of which is $120^{\\circ}$, form an arithmetic progression with a difference of $5^{\\circ}$. Determine the number of sides of the polygon.",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 50669,
        "difficulty": null
      },
      {
        "id": "a6f1a5e9dfa52e8f292fa7eae22f745e058755e5544c93b6cccd29c5d13e52d9",
        "domain": "numina",
        "question": "3. How many nine-digit numbers written using nine different digits 1, 2, 3, 4, 5, 6, 7, 8, 9 are there such that each three-digit number formed by three consecutive digits of that number is divisible by 3?",
        "answer": "1296",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86883,
        "difficulty": null
      },
      {
        "id": "a18af14b92a7d8a21cd8f4dc2e506c36114da6dbf4d7e10697e3a9197412be19",
        "domain": "numina",
        "question": "Zhendarov R.G.\n\nIn a bank, there are 2002 employees. All employees came to the anniversary and were seated at one round table. It is known that the salaries of those sitting next to each other differ by 2 or 3 dollars. What is the greatest possible difference between two salaries of the employees of this bank, given that all salaries are different?\n\n#",
        "answer": "3002",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 52448,
        "difficulty": null
      },
      {
        "id": "1ae9d7a6cb9f2a39ea7fca7d24a88efefacddf32e73d95f1bea66903c1d22b2f",
        "domain": "numina",
        "question": "3-4. There is a piece of chain consisting of 150 links, each weighing 1 g. What is the smallest number of links that need to be unbuckled so that from the resulting parts, all weights of 1 g, 2 g, 3 g, \\(\\ldots, 150\\) g can be formed (an unbuckled link also weighs 1 g)?",
        "answer": "4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 26323,
        "difficulty": null
      },
      {
        "id": "48db77d5a52173127c3c40e162dfef8ce4baf613e95428ec6601d21bfba48df1",
        "domain": "numina",
        "question": "2 [Arithmetic. Mental arithmetic, etc.]\n\nThere are nuts in the boxes. In the first box, there are 6 kg fewer nuts than in the other two together. And in the second box, there are 10 kg fewer nuts than in the other two together. How many nuts are in the third box?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 71738,
        "difficulty": null
      },
      {
        "id": "05ef1e4edffe39d6c14bced86e9cb68eaf99ee56a5bc149da17136377943aac5",
        "domain": "numina",
        "question": "Find all real numbers $x, y, z$ such that\n\n$$\nx+y+z=3, \\quad x^{2}+y^{2}+z^{2}=3, \\quad x^{3}+y^{3}+z^{3}=3\n$$\n\nDistinction between polynomial and polynomial function Here, we explain why it is necessary to make this distinction, starting by defining a polynomial in another way. Here, $\\mathbb{K}=\\mathbb{Q}, \\mathbb{R}, \\mathbb{C}$ or $\\mathbb{Z} / p \\mathbb{Z}$ equipped with the usual addition and multiplication laws.\n\nDefinition. A polynomial with coefficients in $\\mathbb{K}$ is an infinite sequence of elements of $\\mathbb{K}$ that is zero from a certain rank.\n\nExample. For example, $(0,1,2,3,0,0, \\ldots, 0, \\ldots)$ is a polynomial, as is $(0,0, \\ldots, 0, \\ldots)$. However, $(1,1, \\ldots, 1, \\ldots)$ is not one.\n\nDefinition. Let $P=\\left(u_{n}\\right)_{n}$ and $Q=\\left(v_{n}\\right)_{n}$ be two polynomials. We define the polynomial $P+Q$ by the sequence $w_{n}=u_{n}+v_{n}$ (which is indeed zero from a certain rank) and the polynomial $P \\times Q$ by the sequence $\\left(z_{n}\\right)$, where $z_{n}=\\sum_{i+j=n} u_{n} v_{n}$ (verify that $\\left(z_{n}\\right)$ is zero from a certain rank). We identify the elements of $\\mathbb{K}$ with the constant polynomials via the application that associates to an element $\\lambda \\in \\mathbb{K}$ the polynomial $(\\lambda, 0,0, \\ldots, 0, \\ldots)$. Note that this is consistent with the intuitive notion of multiplying a polynomial by an element of $\\mathbb{K}$: if $\\left(u_{n}\\right)$ is a polynomial and $\\lambda \\in \\mathbb{K}$, then the polynomial $\\lambda \\times\\left(u_{n}\\right)$ is the polynomial $\\left(\\lambda u_{n}\\right)$.\n\nWe now introduce the indeterminate $X$.\n\nDefinition. Let us denote by $X$ the polynomial $(0,1,0,0, \\ldots)$.\n\nProposition. Every polynomial $P$ can be expressed in the form $P=\\sum_{i=0}^{n} a_{n} X^{n}$. We denote $P$ or $P(X)$ indifferently to remind that we denote $X$ the indeterminate (we could very well denote it $Y!$ ).\n\nProof. If $P=\\left(a_{0}, a_{1}, a_{2}, \\ldots\\right)$, let $N$ be an integer such that $i \\geq N$ implies $a_{i}=0$. Then $P(X)=a_{0}+a_{1} X+\\cdots+a_{N} x^{N}$. This is an immediate consequence of the definition of $X$ and the multiplication between polynomials.\n\nHere is now the link between polynomial and associated polynomial function. Recall that, for now, a polynomial is just a sequence of numbers that is zero from a certain rank and is not seen as an application.\n\nProposition. Let $P(X)=a_{0}+a_{1} X+\\cdots+a_{n} X^{n} \\in \\mathbb{K}[X]$ be a polynomial. We denote by $\\widetilde{P}$ the application defined by $\\widetilde{P}(x)=a_{0}+a_{1} x+\\cdots+a_{n} x^{n}$ for $x \\in \\mathbb{K}$, which we call the polynomial function associated with $P$. The application $P \\mapsto \\widetilde{P}$ is injective if $\\mathbb{K}$ is infinite. If $\\mathbb{K}$ is finite, this application is not necessarily injective.\n\nProof. First, consider the case where $\\mathbb{K}$ is infinite. Let $P, Q \\in \\mathbb{K}[X]$ such that $\\widetilde{P}=\\widetilde{Q}$. Write $P(X)=\\sum_{i} a_{i} X^{i}$ and $Q(X)=\\sum_{i} b_{i} X^{i}$. Then the polynomial $P(x)-Q(x)$, in the sense of the previous sections, has an infinite number of roots, so it is zero. Therefore, $a_{i}=b_{i}$ for all $i$.\n\nHowever, in the case where $\\mathbb{K}$ is finite, the previous reasoning does not apply. Moreover, let us exhibit a counterexample. Consider $\\mathbb{K}=\\mathbb{Z} / p \\mathbb{Z}$ and $P(X)=X^{p}-X$. According to Fermat's little theorem, for all $x \\in \\mathbb{K}$, we have $P(x)=0$. Thus, $P$ is not the zero polynomial, but the two associated polynomial functions are the same.\n\nIn other words, when $\\mathbb{K}$ is infinite (for example, $\\mathbb{K}=\\mathbb{Q}, \\mathbb{R}, \\mathbb{C}$, which explains why we did not lose generality in the first sections), we can talk without distinction about a polynomial or the associated polynomial function. However, in other cases, one must be very careful!\n\nElements of the answer to the exercises",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 82888,
        "difficulty": null
      },
      {
        "id": "01c3b91ca9a35f92a94f769b0b298a1be1918ccd0b6ff39c0a51af01d96c5cc5",
        "domain": "numina",
        "question": "6. (10 points) Car A and Car B depart from City A to City B, which is 300 kilometers away. It is known that Car A departs 1 hour later than Car B, but arrives 1 hour earlier at City B. Then, Car A catches up with Car B at a distance of $\\qquad$ kilometers from City B.",
        "answer": "150",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92349,
        "difficulty": null
      },
      {
        "id": "6086eb858ebd94b50c51540fcdbdadb865023436600cc5039a84ca03feb74c78",
        "domain": "numina",
        "question": "Problem 8-5. 8 8-5. A divisor of a natural number is called proper if it is different from 1 and the number itself. Find all natural numbers for which the difference between the sum of the two largest proper divisors and the sum of the two smallest proper divisors is a prime number.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 20144,
        "difficulty": null
      },
      {
        "id": "8b76dec663ec4e0b41214034bb54ecb281db8a8df8324c20de988b133377e8f1",
        "domain": "numina",
        "question": "2. Given two linear functions $f(x)$ and $g(x)$ such that the graphs $y=f(x)$ and $y=g(x)$ are parallel lines, not parallel to the coordinate axes. Find the minimum value of the function $(g(x))^{2}+$ $2 f(x)$, if the minimum value of the function $(f(x))^{2}+2 g(x)$ is 5.",
        "answer": "-7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3311,
        "difficulty": null
      },
      {
        "id": "d219797129f0b357380f9d882482a2ffd3a33e1d12ba5efcc532b1b7d215eb8c",
        "domain": "numina",
        "question": "14. On a rectangular chessboard filled with chess pieces, two players, A and B, take turns to pick up the pieces, following these rules: in a certain row or column, take any continuously placed pieces (i.e., cannot pick across empty spaces), not allowed to skip a turn, and cannot pick from multiple rows (columns) at once. The game ends when all pieces on the board are taken. The player who takes the last piece wins.\nFaced with the chessboard as shown in the figure, the first player has a winning strategy. The first player\n\nshould take\n\n(all correct solutions), to ensure victory.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 134457,
        "difficulty": null
      },
      {
        "id": "70b667fcd57bd6649593f0659e24bead0263a8a0354a047af61fd04ceb945543",
        "domain": "numina",
        "question": "Example 13 Find the maximum value of the function $f(x)=\\sqrt{10 x-9-x^{2}}+\\sqrt{68 x-256-x^{2}}$.",
        "answer": "3\\sqrt{35}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117618,
        "difficulty": null
      },
      {
        "id": "03cd1ad42e58a1eb6fbefe12d35d564f02ea9f3236a80f6c57284ac137879b0d",
        "domain": "numina",
        "question": "5. How many days have passed from March 19, 1990 to March 23, 1996 inclusive",
        "answer": "2196",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 42703,
        "difficulty": null
      },
      {
        "id": "6453f6a6ee745c23e8047ee82f99b47d9554e9ec86409b5220d358c1155c774f",
        "domain": "numina",
        "question": "7.3. How many five-digit natural numbers exist, each of which has adjacent digits with different parity",
        "answer": "5625",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 22051,
        "difficulty": null
      },
      {
        "id": "5e7d3be9a44d313525626d466f48754e37a5d9fccd501005e56edb0207f5b9fa",
        "domain": "numina",
        "question": "1. Let the right vertex of the ellipse $\\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$ be $A$, the upper vertex be $B$, and the left focus be $F$. If $\\angle A B F=90^{\\circ}$, then the eccentricity of the ellipse is $\\qquad$.",
        "answer": "\\frac{\\sqrt{5}-1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105110,
        "difficulty": null
      },
      {
        "id": "5e71a5dfa4bf6e54b8353903c5b3aa5281988447a9fadedad580eae9f87c2741",
        "domain": "numina",
        "question": "6、Let $x$ be an acute angle, then the maximum value of the function $y=\\sin x \\sin 2 x$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly. \n\n6、Let $x$ be an acute angle, then the maximum value of the function $y=\\sin x \\sin 2 x$ is $\\qquad$",
        "answer": "\\frac{4\\sqrt{3}}{9}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142985,
        "difficulty": null
      },
      {
        "id": "1b285d9a18206b548540d706b9846fb1c792aced1dff02a406caaded7e842568",
        "domain": "numina",
        "question": "Determine the smallest positive integer $n$ for which there exists a polynomial\n\n$$\nP(X)=a_{2 n} X^{2 n}+a_{2 n-1} X^{2 n-1}+\\ldots+a_{1} X+a_{0}\n$$\n\nwith real coefficients that satisfies the following two properties:\n\n- For $i=0,1, \\ldots, 2 n$, we have $2014 \\leq a_{i} \\leq 2015$.\n- There exists a real number $\\xi$ such that $P(\\xi)=0$.",
        "answer": "2014",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 195894,
        "difficulty": null
      },
      {
        "id": "da5743a220aed926f457ff63b5703df696b94efdbd3b642d921901abc869ae1c",
        "domain": "numina",
        "question": "12. Given a moving point $P(x, y)$ satisfies the quadratic equation $10 x-2 x y-2 y+1=0$, then the eccentricity of this quadratic curve is $\\qquad$.",
        "answer": "\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 135138,
        "difficulty": null
      },
      {
        "id": "99bf55103acb29e0ea7c9afb0d1b54996b449b665265acd3721c74dcc69a1e41",
        "domain": "numina",
        "question": "2. [5] Evaluate the sum\n$$\n1-2+3-4+\\cdots+2007-2008 .\n$$",
        "answer": "-1004",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151040,
        "difficulty": null
      },
      {
        "id": "e8391d46e805904fa58b8ae5c9a15319ec9fc49e61b98ae5f5b6b8ceafe29775",
        "domain": "numina",
        "question": "## Task 3 - 020723\n\nEmil tells: \"My brother Heinz is only half as old as I am. If you multiply the number of his years by itself, you get my father's age. My mother is 5 years younger than my father. All together, we are 85 years old.\"\n\nHow old is Emil? Describe how you found the solution!",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183485,
        "difficulty": null
      },
      {
        "id": "d592fbe4bbc16dc4bed101bd4f33abf616e749094e2e8c41fdd947c542316163",
        "domain": "numina",
        "question": "6. Given $\\overrightarrow{O A}$ and $\\overrightarrow{O B}$ are non-zero and non-collinear vectors. Let $\\overrightarrow{O C}=\\frac{1}{1+r} \\overrightarrow{O A}+\\frac{r}{1+r} \\overrightarrow{O B}$. Define the set of points\n$$\nM=\\left\\{K \\left\\lvert\\, \\frac{\\mid \\overrightarrow{K A} \\cdot \\overrightarrow{K C}}{|\\overrightarrow{K A}|}=\\frac{\\overrightarrow{K B} \\cdot \\overrightarrow{K C}}{|\\overrightarrow{K B}|}\\right.\\right\\} .\n$$\n\nWhen $K_{1} 、 K_{2} \\in M$, if for any $r \\geqslant 2$, the inequality $\\left|\\overrightarrow{K_{1} K_{2}}\\right| \\leqslant c|\\overrightarrow{A B}|$ always holds, then the minimum value of the real number $c$ is $\\qquad$",
        "answer": "\\frac{4}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98684,
        "difficulty": null
      },
      {
        "id": "12626cd5967e8a10f2bd17489065b3589b0af8b4a6042d363f04a3ab73c21330",
        "domain": "numina",
        "question": "7.2. Three cyclists leave the city. The speed of the first one is 12 km/h, the second one is 16 km/h, and the third one is 24 km/h. It is known that the first one was cycling exactly when the second and third were resting (standing still), and that at no time did two cyclists cycle simultaneously. It is also known that in 3 hours, all three covered the same distance. Find this distance.",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12735,
        "difficulty": null
      },
      {
        "id": "d30a640b1ff47aba35cad6cd52574f9b95274974ee0193db62a66ecebf523784",
        "domain": "numina",
        "question": "1. If the 5th term in the expansion of $\\left(x \\sqrt{x}-\\frac{1}{x}\\right)^{6}$ is $\\frac{15}{2}$, then $\\lim _{n \\rightarrow \\infty}\\left(x^{-1}+x^{-2}+\\cdots+x^{-n}\\right)=$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109210,
        "difficulty": null
      },
      {
        "id": "795084d119861217ca99709aa19f5a75d741dfc66be0f898626f2bc927b930c5",
        "domain": "numina",
        "question": "1. The picture shows several circles connected by segments. Nastya chooses a natural number \\( n \\) and places different natural numbers in the circles so that for all these numbers the following property holds:\n\nif numbers \\( a \\) and \\( b \\) are not connected by a segment, then the difference \\( a - b \\) must be coprime with \\( n \\), and if they are connected, then \\( a - b \\) and \\( n \\) must have a common natural divisor greater than 1. For what smallest \\( n \\) does such an arrangement exist?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6bce8335c6b594326f89g-15.jpg?height=271&width=263&top_left_y=601&top_left_x=908)",
        "answer": "385",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 8010,
        "difficulty": null
      },
      {
        "id": "502d765a55b92f86b2af0dc1a345d51edc62f15d66dde10cea29f72ffecea77a",
        "domain": "numina",
        "question": "13. (10 points) A class of 30 students goes to the movies, and their student IDs are $1, 2, \\cdots, 30$; the movie tickets they hold are exactly the 1st, 2nd, ..., 30th seats in a row. Now, the movie tickets are to be distributed to these students according to the following requirements: for any two students, A and B, if A's student ID can be divided by B's student ID, then A's movie ticket number can also be divided by B's movie ticket number. How many different ways are there to distribute the movie tickets? $\\qquad$",
        "answer": "48",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119079,
        "difficulty": null
      },
      {
        "id": "8b221719213bcee55122322b47194841775ca6cf2ed82ff285097ab1143714f2",
        "domain": "numina",
        "question": "5. A product that initially contained $98\\%$ water dried over some time and began to contain $95\\%$ water. By what factor did it shrink (i.e., reduce its weight)?",
        "answer": "2.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4920,
        "difficulty": null
      },
      {
        "id": "aef17522e59e39b0276c0cc9e42aa8983ab731e792b3498bd812ec1910bc0383",
        "domain": "numina",
        "question": "4. Dad, Mom, the guest, and I, four people, are sitting around a round table having tea. If we only consider the person on each person's left, how many different seating arrangements are there?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 124865,
        "difficulty": null
      },
      {
        "id": "fc77b17623c9338c8a36749ec79b1e39c36bb48b4bfc254cad5d70c250daad49",
        "domain": "numina",
        "question": "# 3. CONDITION\n\nGiven a cube $A B C D A_{1} B_{1} C_{1} D_{1}$ with edge 1. A line $l$ passes through point $E$, the midpoint of edge $C_{1} D_{1}$, and intersects lines $A D_{1}$ and $A_{1} B$. Find the distance from point $E$ to the point of intersection of line $l$ with line $A_{1} B$.",
        "answer": "1.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13585,
        "difficulty": null
      },
      {
        "id": "fd690623ad09c856d0c374491415527227fc0a0c2921c6da3b9f35d2ac5e9d8a",
        "domain": "numina",
        "question": "## Task B-2.3.\n\nOn the side $\\overline{A B}$ of triangle $A B C$, a point $E$ is given such that $|A E|=1$ and $|E B|=2$. Let point $D$ be on the side $\\overline{A C}$ such that the segment $\\overline{D E}$ is parallel to the segment $\\overline{B C}$, and point $F$ be on the side $\\overline{B C}$ such that the segment $\\overline{E F}$ is parallel to the segment $\\overline{A C}$. Calculate the ratio of the areas of quadrilateral $C D E F$ and triangle $A B C$.",
        "answer": "\\frac{4}{9}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88676,
        "difficulty": null
      },
      {
        "id": "28b54e017b4d55ddd476c183b3e7ed76ae6a46a6f272d3f07384da90db15ed2f",
        "domain": "numina",
        "question": "G9.4 Find the sum $d$ where\n$$\nd=\\left(\\frac{1}{2}+\\frac{1}{3}+\\frac{1}{4}+\\cdots+\\frac{1}{60}\\right)+\\left(\\frac{2}{3}+\\frac{2}{4}+\\frac{2}{5}+\\cdots+\\frac{2}{60}\\right)+\\left(\\frac{3}{4}+\\frac{3}{5}+\\cdots+\\frac{3}{60}\\right)+\\cdots+\\left(\\frac{58}{59}+\\frac{58}{60}\\right)+\\frac{59}{60}\n$$",
        "answer": "885",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152782,
        "difficulty": null
      },
      {
        "id": "5604aa57abbc6785767bc7be827cd5b038cb73990513620a3c823dd0130ed030",
        "domain": "numina",
        "question": "8. A figure composed of some vertices and edges is called a graph. Coloring the vertices of a graph with different colors, requiring that two vertices connected by the same edge be colored differently, is called vertex coloring of a graph. The usual problem studied in graph vertex coloring is the minimum number of colors needed to complete the coloring, which is called the chromatic number of the graph. The graph below is called the Petersen graph, and the chromatic number of the Petersen graph is _",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117260,
        "difficulty": null
      },
      {
        "id": "2a6cd83b6cf29c6fac3e9f1969e883270e858ed867d1c3367242de98140980a9",
        "domain": "numina",
        "question": "G9.4 If the sum of the interior angles of an $n$-sided convex polygon is $x^{\\circ}$, find $x$.",
        "answer": "3240",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152513,
        "difficulty": null
      },
      {
        "id": "89457306660461a80202afb87345cdbd0556d6df0b885fbd8b7fb35093b6300e",
        "domain": "numina",
        "question": "6. (5 points) There are four natural numbers, their sum is 243. If the first number is increased by 8, the second number is decreased by 8, the third number is multiplied by 8, and the fourth number is divided by 8, then the four resulting numbers are equal. Therefore, the product of the largest and smallest of the original four numbers is . $\\qquad$",
        "answer": "576",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111069,
        "difficulty": null
      },
      {
        "id": "340f179bbbea2a78ecc67f603113fa19c091f5e3a8fc769c7a9f57a1a7d33e82",
        "domain": "numina",
        "question": "Three mountains lie on the same meridian. The spherical distance between the first two is $a^{0}$, and between the second two is $b^{0}$; their heights are $h_{1}, h_{2}, h_{3}$ respectively. Does the second mountain obscure the third mountain when viewed from the first mountain?\n\nNumerical values: $a=1^{\\circ}, b_{0}=1^{\\circ} 30^{\\prime}, h_{1}=2500 \\mathrm{~m}, h_{2}=3000 \\mathrm{~m}, h_{3}=8800 \\mathrm{~m}$ and the radius of the Earth is $6366 \\mathrm{~km}$.",
        "answer": "1500",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171951,
        "difficulty": null
      },
      {
        "id": "2f84f8fed60c0fdc67b0e92ef3eaa48391d0a9c8d7591ae50629ce6592440357",
        "domain": "numina",
        "question": "4. Let $[x]$ denote the greatest integer not exceeding the real number $x$. Calculate: $\\sum_{k=0}^{2019}\\left[\\frac{4^{k}}{5}\\right]=$ $\\qquad$ .",
        "answer": "\\frac{4^{2020}-1}{15}-1010",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114309,
        "difficulty": null
      },
      {
        "id": "18fa8d48391176713e5d30ad7e806e920b761ca9c484c01be2f9ce058860d168",
        "domain": "numina",
        "question": "Consider 9 points in the plane with no alignment. What is the minimum value of $n$ such that if we color $n$ edges connecting two of the points in red or blue, we are sure to have a monochromatic triangle regardless of the coloring?\n\n(IMO 1992)",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 82996,
        "difficulty": null
      },
      {
        "id": "6ca92786032df6ba7057f61541c50969cb918458d7d9c4ab6ee0bbae99744578",
        "domain": "numina",
        "question": "3. (7 points) There are 8 blue, 7 red, and 12 white light bulbs. In how many ways can they be arranged into a garland (using all the bulbs) so that no two white bulbs are adjacent?",
        "answer": "11711700",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19410,
        "difficulty": null
      },
      {
        "id": "b060db009fbba3750a632166971b723dad00d6ad1a3246f77557dec6b39935f6",
        "domain": "numina",
        "question": "The fifth question: If an infinite sequence of positive real numbers $\\left\\{x_{n}\\right\\}$ satisfies: $x_{0}=1, x_{i} \\geq x_{i+1}(i \\in N)$, then the sequence is called a “good sequence”. Find the smallest constant $c$ such that there exists a “good sequence” $\\left\\{x_{n}\\right\\}$ satisfying: $\\sum_{i=0}^{n} \\frac{x_{i}^{2}}{x_{i+1}} \\leq c$ for any positive integer $n$.",
        "answer": "4",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 111270,
        "difficulty": null
      },
      {
        "id": "b4bc846462fd1ab72e85d179cc64ce6ab41ab18fa6868decb588f5d3f0770cfe",
        "domain": "numina",
        "question": "5. Drop perpendiculars $D D_{1}, D D_{2}, D D_{3}$ from point $D$ to the planes $S B C$, $S A C$, and $S A B$ respectively. Let $D D_{1}=x, D D_{2}=y, D D_{3}=z$. According to the condition, we form the system of equations\n\n$$\n\\left\\{\\begin{array}{l}\ny^{2}+z^{2}=5 \\\\\nx^{2}+z^{2}=13 \\\\\nx^{2}+y^{2}=10\n\\end{array}\\right.\n$$\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0013fadf99b0b72b3281g-9.jpg?height=526&width=630&top_left_y=95&top_left_x=1848)\n\nFig. 6:\n\nFrom here, we find $x=3, y=1, z=2$. Let the lengths of the edges $S A, S B$, and $S C$ be $a, b$, and $c$ respectively. Since points $A, B, C$, and $D$ lie in the same plane, the relation $\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}=1$ holds.\n\nUsing the inequality between the arithmetic mean and the geometric mean for three variables, we get:\n\n$$\n\\begin{aligned}\n& \\frac{\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}}{3} \\geqslant \\sqrt[3]{\\frac{3}{a} \\cdot \\frac{1}{b} \\cdot \\frac{2}{c}}=\\sqrt[3]{\\frac{6}{a b c}} \\Longleftrightarrow \\\\\n& \\Longleftrightarrow 1=\\left(\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}\\right)^{3} \\geqslant \\frac{6 \\cdot 27}{a b c} \\Longleftrightarrow a b c \\geqslant 6 \\cdot 27\n\\end{aligned}\n$$\n\nwith equality holding when $\\frac{3}{a}=\\frac{1}{b}=\\frac{2}{c}=\\frac{1}{3}$. The volume of the pyramid $V=\\frac{a b c}{6}$, so $V \\geqslant 27$. Equality holds when $a=9, b=3, c=6$.",
        "answer": "27",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5195,
        "difficulty": null
      },
      {
        "id": "b89853c3a456a10c1cfc13bc28bc433214cc393bf460f37980cfaf3696237ca7",
        "domain": "numina",
        "question": "8. In a right trapezoid, the upper base is shorter than the lower base. If the volume of the solid of revolution obtained by rotating it around its lower base is $80 \\pi$,\n\nthe volume of the solid of revolution obtained by rotating it around its upper base is $112 \\pi$, and the volume of the solid of revolution obtained by rotating it around its right leg is $156 \\pi$, then the perimeter of the right trapezoid is",
        "answer": "16+2\\sqrt{13}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 91525,
        "difficulty": null
      },
      {
        "id": "ef92b8092980785613c58abfc1616dca6b80759ad924f0ebe6b5b65f6f40bb33",
        "domain": "numina",
        "question": "433. Gifted Paintings. A wealthy collector had 10 valuable paintings. He wanted to make a gift to a museum, but the collector couldn't figure out how many gift options he had: after all, he could give any one painting, any two, any three paintings, and so on, he could even give all ten paintings.\n\nThe reader might think that answering this question would require a long and tedious calculation; however, I will provide a small rule that allows one to answer in all similar cases without any difficulties and thankless work.",
        "answer": "1023",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 41348,
        "difficulty": null
      },
      {
        "id": "6e42056e1d40c34f047d47c75ec7c648e312095ddea4a59d28886fa30967e074",
        "domain": "numina",
        "question": "2. Simplify the expression $x^{(x+1)^{2}} \\cdot\\left(x^{x-1}\\right)^{x+1}: \\frac{x^{x^{2}}}{x^{-2 x}}$ and then calculate the value of the expression for $x=\\sqrt{2}$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163771,
        "difficulty": null
      },
      {
        "id": "da69dac2364c7afe47ac57b46ca0ead58025d46113013682ebd9c3492bbfdc8c",
        "domain": "numina",
        "question": "3. A $4 \\times 4 \\times h$ rectangular prism can hold 8 small spheres with a radius of 1 and 1 large sphere with a radius of 2. Then the minimum value of $h$ is $\\qquad$ .",
        "answer": "2+2\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124959,
        "difficulty": null
      },
      {
        "id": "c65a017743391ae3e5bb3523ab4e2ac2b58eb44542b2ce82ce0b3670e65bbaa1",
        "domain": "numina",
        "question": "3. In triangle $ABC (\\overline{BC} > \\overline{AC})$, the angles $\\alpha = \\measuredangle BAC$ and $\\beta = \\measuredangle ABC$ differ by $30^{\\circ}$. If point $D$ is chosen on side $BC$ such that $\\overline{AC} = \\overline{CD}$, calculate $\\measuredangle DAB$.",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189296,
        "difficulty": null
      },
      {
        "id": "102b52b62d7a4163d1df488bdc9ab59337adf50e39d4f7998f5549b9a4515c3b",
        "domain": "numina",
        "question": "10. (10 points) In a math test at Sheep Village Elementary School's fourth grade, the test consists of 15 questions. If Xiaoxixi, Xiaobeibe, Xiaomeimei, and Xiaolan all answered 11, 12, 13, and 14 questions correctly, respectively, then the minimum number of questions they all answered correctly is $\\qquad$ questions.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 100571,
        "difficulty": null
      },
      {
        "id": "447263584e67672b9249a5ccdf13dbb60d0f31a2a5b542b5aeb49a7aeb7ff3bc",
        "domain": "numina",
        "question": "4. Given three non-zero real numbers $a, b, c$ such that the equations: $a x^{2}+b x+c=0, b x^{2}+c x+a=0, c x^{2}+a x+b=0$ each have two roots. How many of the roots of these equations can be negative?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15248,
        "difficulty": null
      },
      {
        "id": "d16a52478267c7dd6c71c93c7c555d258a7df5cd76216bb60084d7df6a6a0b22",
        "domain": "numina",
        "question": "19. (6 points) Xiao Ming puts 127 Go stones into several bags. No matter how many stones a child wants (not exceeding 127 stones), Xiao Ming can satisfy the request by taking out a few bags. How many bags does Xiao Ming need to prepare at least?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 119215,
        "difficulty": null
      },
      {
        "id": "aa40ba62d005e5162ace24e89dc5a48baf688d351091f719ab36dee20d5c4cc1",
        "domain": "numina",
        "question": "Problem 4.7. Denis threw darts at four identical dartboards: he threw exactly three darts at each board, where they landed is shown in the figure. On the first board, he scored 30 points, on the second - 38 points, on the third - 41 points. How many points did he score on the fourth board? (For hitting each specific zone - ring or central field - a certain number of points is awarded.)\n![](https://cdn.mathpix.com/cropped/2024_05_06_c49ecf27446af41a8bcdg-06.jpg?height=260&width=1136&top_left_y=960&top_left_x=156)",
        "answer": "34",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17839,
        "difficulty": null
      },
      {
        "id": "e298f55ad3066e99c8b344cdb02fc34f348a141e792fa73ab179cfcd02e7672e",
        "domain": "numina",
        "question": "4. Xiaoming formed a sequence using the four digits $2, 0, 1, 6$ (continuously adding these four digits in this order to the end of the number): $2,20,201,2016,20162,201620,2016201,20162016,201620162, \\ldots$. In this sequence, the number of prime numbers is $\\qquad$.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118928,
        "difficulty": null
      },
      {
        "id": "b0e7d38e414dd30ad862afa527ea70260e2cd98fb2099fd3c1c19649f525ac7d",
        "domain": "numina",
        "question": "1. How many positive integers less than 2019 are divisible by either 18 or 21 , but not both?",
        "answer": "176",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153969,
        "difficulty": null
      },
      {
        "id": "9aa78e43bb20b7088fdc9c227762f077b00d383d8e3be4e9ee3ab446b2b67166",
        "domain": "numina",
        "question": "Example 9 (1) When $0 \\leqslant x \\leqslant 1$, find the range of the function\n$f(x)=(\\sqrt{1+x}+\\sqrt{1+x}+2) \\cdot\\left(\\sqrt{1-x^{2}}+1\\right)$.\n(2) Prove that when $0 \\leqslant x \\leqslant 1$, there exists a positive number $\\beta$ such that the inequality $\\sqrt{1+x}+\\sqrt{1-x} \\leqslant 2-\\frac{x^{a}}{\\beta}$ holds for the smallest positive number $\\alpha=2$, and find the smallest positive number $\\beta$ at this time.",
        "answer": "4",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 110037,
        "difficulty": null
      },
      {
        "id": "55ebc1e7cdc5e4606fbd904c75c134971ed6ce72fed3b5bf3d66a3edf2cbdfe6",
        "domain": "numina",
        "question": "15.1.1 * Let $a_{n}=6^{n}+8^{n}$, determine the remainder when $a_{83}$ is divided by 49.",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 140925,
        "difficulty": null
      },
      {
        "id": "6f909c49f18d7fccbcaef9331ecdaf4c073163b99b97f13666fda71a7615cb79",
        "domain": "numina",
        "question": "Problem 11.1. Masha lives in apartment No. 290, which is located in the 4th entrance of a 17-story building. On which floor does Masha live? (The number of apartments is the same in all entrances of the building on all 17 floors; apartment numbers start from 1.$)$",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13688,
        "difficulty": null
      },
      {
        "id": "3af0041a55166ee972fdf0d1ecd2d41d113786af761db33a684efbfdb985ff4b",
        "domain": "numina",
        "question": "7. Given $1 \\leqslant x^{2}+y^{2} \\leqslant 4$, then the sum of the maximum and minimum values of $x^{2}-x y+y^{2}$ is\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "6\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108887,
        "difficulty": null
      },
      {
        "id": "afb51a80b08f988f9fe220512a49364ae0e31e605aaa3dc9e73698eb67a9a59f",
        "domain": "numina",
        "question": "## Task B-1.2.\n\nTwo pipes fill a pool in 4 hours. One of these pipes fills the pool by itself in 5 hours and 15 minutes. How long does it take for the other pipe to fill the pool by itself?",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88221,
        "difficulty": null
      },
      {
        "id": "99961856a93a2cda34404595bb594f036b88a5bae50529f7756314ca2d36b945",
        "domain": "numina",
        "question": "73. Using $0, 2, 6, 8$, we can form $\\qquad$ number of three-digit numbers without repeated digits.",
        "answer": "18",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 95305,
        "difficulty": null
      },
      {
        "id": "78b774829729cadceacf649bb155089627d012ecbec15d92662ae766d0555e85",
        "domain": "numina",
        "question": "In a certain kingdom, there were 32 knights. Some of them were vassals of others (a vassal could have only one suzerain, and a suzerain was always richer than his vassal). A knight who had at least four vassals bore the title of baron. What is the maximum number of barons that could be under these conditions?\n\n(The kingdom had a law: \"a vassal of my vassal is not my vassal\".)\n\n#",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56800,
        "difficulty": null
      },
      {
        "id": "ced909d6ffe4d27d8f4e97821d948d8a3e419f931c12c89910de3131834af589",
        "domain": "numina",
        "question": "3. Given a positive real number $x$ satisfies\n$$\nx^{3}+x^{-3}+x^{6}+x^{-6}=2754 \\text {. }\n$$\n\nThen $x+\\frac{1}{x}=$ $\\qquad$ .",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93659,
        "difficulty": null
      },
      {
        "id": "c8cf0b2fc0711895203f668adb2700a3b119b81901efa040ccc6d38b559a0238",
        "domain": "numina",
        "question": "423. Can the sum of a finite number of terms of a geometric progression be zero?",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 40192,
        "difficulty": null
      },
      {
        "id": "2e6c22030d90fe2112b473052093644a485747849da9df1d20168e599268a6cf",
        "domain": "numina",
        "question": "1. (7 points) In an $8 \\times 8$ frame with a width of 2 cells (see figure), there are a total of 48 cells. How many cells are in a $254 \\times 254$ frame with a width of 2 cells?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_af60849ded79da38d4cfg-1.jpg?height=288&width=280&top_left_y=661&top_left_x=891)",
        "answer": "2016",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13376,
        "difficulty": null
      },
      {
        "id": "1fd9c91aabf7a7527515f153c69d93f09c41877d1863389b114cf32957511b07",
        "domain": "numina",
        "question": "Example 3 In the Cartesian coordinate system $x O y$, given two points $M(-1,2)$ and $N(1,4)$, point $P$ moves on the $x$-axis. When $\\angle M P N$ takes the maximum value, find the x-coordinate of point $P$.\n(2004 National High School Mathematics Competition Problem)",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137533,
        "difficulty": null
      },
      {
        "id": "807f25b2f97cb6dffe2a510b314e085af7f3fcba2b7400482ca06bf375e07e2e",
        "domain": "numina",
        "question": "A rectangle with sides of $15 \\mathrm{~cm}$ and $20 \\mathrm{~cm}$ is rotated about one of its diagonals. What is the volume of the resulting solid of revolution?",
        "answer": "5238.85",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 169801,
        "difficulty": null
      },
      {
        "id": "99e8d83920362de0ad445553c139f58eca2ea6f3aedbd326d25faaf292293968",
        "domain": "numina",
        "question": "9. Find the maximum value of the function $f(x)=9 \\sin x+12 \\cos x$.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19189,
        "difficulty": null
      },
      {
        "id": "fc7d97235396fc0d257a6f429442a9618fbacf8a7d0255f98991e1a0b3da7fc7",
        "domain": "numina",
        "question": "54. As shown in the figure, a square is divided into two identical L-shapes. If the two L-shapes can form a rectangle with a perimeter of 26, find the area of this square.",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142513,
        "difficulty": null
      },
      {
        "id": "70c91ca45bc8cc0ee1a4bf9a001433f334ba3a764b56c0751c3df32049408c53",
        "domain": "numina",
        "question": "6. Given three points $A, B, C$ on a plane satisfying $|\\overrightarrow{A B}|=3,|\\overrightarrow{B C}|=4,|\\overrightarrow{C A}|=5$, then $\\overrightarrow{A B} \\cdot \\overrightarrow{B C}+\\overrightarrow{B C} \\cdot \\overrightarrow{C A}+\\overrightarrow{C A} \\cdot \\overrightarrow{A B}$ is equal to",
        "answer": "-25",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137414,
        "difficulty": null
      },
      {
        "id": "1ca799cb4c96a9e552ad989cde74296013a6d65f77b547cdbbdbe3dbf47305e7",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$$\n\\lim _{x \\rightarrow 0} \\frac{\\sqrt{1+x \\sin x}-1}{e^{x^{2}}-1}\n$$",
        "answer": "\\frac{1}{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47724,
        "difficulty": null
      },
      {
        "id": "121a9273152bbe65afb8948c68e4adec165087b9176cbc4ed8b828a5bef80ed9",
        "domain": "numina",
        "question": "6. The reading on a factory's electricity meter is 52222 kilowatts. After several days, the meter reading (a five-digit number) again shows four identical digits. How much electricity, in kilowatts, did the factory use at least in these days?\n\n untranslated portion: $\\qquad$",
        "answer": "333",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 97559,
        "difficulty": null
      },
      {
        "id": "eb15bc1df0442b442547c96ce7df68491dd96dcdf7fa92d05be23530bf40bc97",
        "domain": "numina",
        "question": "7. There are 5 power poles, 2 of which will leak electricity. There are 5 birds each randomly choosing one of the poles to rest (if a bird touches a leaking pole, it will be stunned by the electric shock and fall to the ground). The probability that only 2 of the poles have birds on them is $\\qquad$",
        "answer": "\\frac{342}{625}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 97952,
        "difficulty": null
      },
      {
        "id": "572b9628e4432f200e61bd6fee2c6b21355047a0ac905a2d4290c2ba352fe809",
        "domain": "numina",
        "question": "Let's construct a triangle, given one side and two heights (2 cases).\n\n\n\n\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\nNote: The provided text is already in English, so no translation is needed for the second part of the instruction.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174677,
        "difficulty": null
      },
      {
        "id": "faa1f67a36f54db03b90802bf8459b9fed1951a93ca74a014a54bc86bd97b1ee",
        "domain": "numina",
        "question": "145. Find $\\lim _{x \\rightarrow 0} \\frac{1-\\cos 8 x}{2 x^{2}}$.",
        "answer": "16",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31954,
        "difficulty": null
      },
      {
        "id": "261b859b94876fcdf67864e850b485bad3629b88da456f2785b85a110e184893",
        "domain": "numina",
        "question": "[ Properties and characteristics of an isosceles triangle. ]\n\nIn trapezoid $ABCD$ ($AD$ is the larger base), diagonal $AC$ is perpendicular to side $CD$ and bisects angle $BAD$. It is known that $\\angle CDA = 60^\\circ$, and the perimeter of the trapezoid is 2. Find $AD$.",
        "answer": "\\frac{4}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63292,
        "difficulty": null
      },
      {
        "id": "eb40951bf24d41b045be7c55c4942e22c4e063de6f9bb7ab03d0c46e48527bb4",
        "domain": "numina",
        "question": "Problem 4.8. The figure shows a road map between the houses of five children. The shortest distance by road from Asya to Galia is 12 km, from Galia to Borya - 10 km, from Asya to Borya 8 km, from Dasha to Galia - 15 km, from Vasya to Galia - 17 km. How many kilometers is the shortest distance by road from Dasha to Vasya?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-07.jpg?height=531&width=872&top_left_y=1073&top_left_x=291)",
        "answer": "18",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17348,
        "difficulty": null
      },
      {
        "id": "2a23149ac46b61f57317a0fa4f3bd5eba22782fd9aa5406526037b07bbe1006e",
        "domain": "numina",
        "question": "63. A large batch of car tires contains $1.5\\%$ defects. What should be the volume of a random sample so that the probability of finding at least one defective car tire in it would be more than $0.92?$",
        "answer": "168",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 31689,
        "difficulty": null
      },
      {
        "id": "075b7dea42b41356f690a6303b5695698980c3418d2993853c39b5f2273f3bdd",
        "domain": "numina",
        "question": "Find all natural numbers $x$, satisfying the conditions: the product of the digits of the number $x$ is equal to $44x - 86868$, and the sum of the digits is a cube of a natural number.",
        "answer": "1989",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 71966,
        "difficulty": null
      },
      {
        "id": "f9d6145893c330be2fd0df2c1e8fe799acf50edb282fcbe324e255c870b13564",
        "domain": "numina",
        "question": "11.145. The lateral faces of a triangular pyramid are mutually perpendicular, and their areas are equal to $a^{2}, b^{2}$ and $c^{2}$. Determine the volume of the pyramid.",
        "answer": "\\frac{1}{3}\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 49002,
        "difficulty": null
      },
      {
        "id": "bb7bad9d6422c761955923b87f662588695ee7e73ad6b91ead1994cd6a996160",
        "domain": "numina",
        "question": "Ex. 72. The height of a right triangle, dropped to the hypotenuse, is $h$. What is the minimum length that the median, which bisects the larger leg, can have?",
        "answer": "\\frac{3}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 41897,
        "difficulty": null
      },
      {
        "id": "35100a610cdbedf85cb970edfc1304b62ea70377be0ba19c483698ee5c7a05ca",
        "domain": "numina",
        "question": "29. Odd integers starting from 1 are grouped as follows: (1), $(3,5),(7,9,11),(13,15,17,19), \\ldots$. where the $n$-th group consists of $n$ odd integers. How many odd integers are in the same group which 2007 belongs to?",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160241,
        "difficulty": null
      },
      {
        "id": "bf34b355f0161d5fdbe3c8203ef856532f446797f0428b010322f8f358ef1d60",
        "domain": "numina",
        "question": "Example 2. Five fair coins are tossed. Find the probability that: exactly 2 heads appeared; more than one head appeared.",
        "answer": "0.8125",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 33048,
        "difficulty": null
      },
      {
        "id": "63e315cb1e3ac1ac26d8d4c460e1016c6132adeebfc2062aa44bbea0c812b190",
        "domain": "numina",
        "question": "4. Determine the minimum value of the expression $E(a, b)=a+b+\\frac{1}{a b}$, where $a, b>0$ and $a+b \\leq 1$.\n\nRomeo Ilie",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179021,
        "difficulty": null
      },
      {
        "id": "880d9fb412eaf754e2c92195b2dfed1740ac5da4790f3cccbce8b011e16c81d4",
        "domain": "numina",
        "question": "5. In a circle, a diameter $A B$ and a chord $C D$ parallel to it were drawn such that the distance between them is half the radius of this circle (see figure). Find the angle $CAB$.",
        "answer": "75",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13213,
        "difficulty": null
      },
      {
        "id": "4910d942666dffe0392fc4f4bdf8c4aeca6de35ad4314e99a877596a3b01e48f",
        "domain": "numina",
        "question": "504. Find the area of a right triangle, both legs of which have a length of 1.",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 43101,
        "difficulty": null
      },
      {
        "id": "c92add1869b499417d75581569f36c0dfa6e193e516a31ef3a16c72f99c9e16e",
        "domain": "numina",
        "question": "How many positive integer solutions does the following equation have?\n\n$$\n\\left[\\frac{x}{10}\\right]=\\left[\\frac{x}{11}\\right]+1\n$$",
        "answer": "110",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 167848,
        "difficulty": null
      },
      {
        "id": "86525d7fd3a3cbeb95a93920d8fd72e20d4ec4d2dcb46a06ce333381ac9961c1",
        "domain": "numina",
        "question": "4. In $\\triangle A B C$, the sides opposite to $\\angle A$, $\\angle B$, and $\\angle C$ are denoted as $a$, $b$, and $c$ respectively. If $b^{2}=a^{2}+c^{2}-a c$, and $c-a$ equals the height $h$ from vertex $A$ to side $A C$, then $\\sin \\frac{C-A}{2}=$ $\\qquad$",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133556,
        "difficulty": null
      },
      {
        "id": "0f5988a0105e211d0971090a43ad44c730b5e03e8b7f6214e8e8404041ce3406",
        "domain": "numina",
        "question": "2. $y=\\arcsin ^{2} x-2 \\arcsin x-2, y_{\\max }=$ $\\qquad$",
        "answer": "\\frac{\\pi^{2}}{4}+\\pi-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125945,
        "difficulty": null
      },
      {
        "id": "58a4bc1dc4b9e418249e9a8aab1b70e9fe4f1174240164d29792b8ed8fa7e0bb",
        "domain": "numina",
        "question": "1. Find the probability that the townspeople win if there are initially two townspeople and one goon.",
        "answer": "\\frac{1}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151549,
        "difficulty": null
      },
      {
        "id": "453f103c6e32d0909f3551a94446b778281fe67fa7678ee4133802116d950044",
        "domain": "numina",
        "question": "G4.1 In Figure $1, A B C D$ is a parallelogram, $B E \\perp C D, B F \\perp A D$, $C E=2 \\mathrm{~cm}, D F=1 \\mathrm{~cm}$ and $\\angle E B F=60^{\\circ}$.\nIf the area of the parallelogram $A B C D$ is $R \\mathrm{~cm}^{2}$, find the value of $R$.",
        "answer": "12\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153072,
        "difficulty": null
      },
      {
        "id": "979a795d290b55c69d803df44ddb47b617498a1e621415ba6fb72d99266bcc34",
        "domain": "numina",
        "question": "11. We call a four-digit number a \"Mid-ring number\" if it has the following characteristics: (1) all four digits are different; (2) the thousand's digit is neither the largest nor the smallest of these four digits; (3) the unit's digit is not the smallest of these four digits. There are $\\qquad$ such \"Mid-ring numbers\".",
        "answer": "1680",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 122145,
        "difficulty": null
      },
      {
        "id": "88f310f93c0f634d05292bbfed2075ce0b17cb5be02e42d7f25093df589f8f78",
        "domain": "numina",
        "question": "1. Let $ABCD$ be an isosceles trapezoid ($AD \\| BC$) with an angle $\\frac{\\pi}{3}$ at the longer base $AD$ and diagonal $\\overline{AC}=\\sqrt{3}$. A point $M$ is at a distance of 1 from $A$ and at a distance of 3 from $D$. Find $\\overline{MC}$.",
        "answer": "\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187349,
        "difficulty": null
      },
      {
        "id": "d16161bbbed038ee2b79edc749b195fc801669a9db882d42b152a31a9ce4c503",
        "domain": "numina",
        "question": "7. [6] Rachel has two indistinguishable tokens, and places them on the first and second square of a $1 \\times 6$ grid of squares, She can move the pieces in two ways:\n- If a token has free square in front of it, then she can move this token one square to the right\n- If the square immediately to the right of a token is occupied by the other token, then she can \"leapfrog\" the first token; she moves the first token two squares to the right, over the other token, so that it is on the square immediately to the right of the other token.\nIf a token reaches the 6th square, then it cannot move forward any more, and Rachel must move the other one until it reaches the 5 th square. How many different sequences of moves for the tokens can Rachel make so that the two tokens end up on the 5 th square and the 6 th square?",
        "answer": "42",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151629,
        "difficulty": null
      },
      {
        "id": "0c6542fe91441413019102c131e47e879a17307fb529d89dd353c2375697416d",
        "domain": "numina",
        "question": "In a right-angled triangle $A B C$, from vertices $B$ and $C$, one point each starts moving simultaneously towards point $A$ along the hypotenuse $B A=c=85$ m and the leg $C A=b=75$ m, respectively. The points move at speeds of $8.5 \\mathrm{~m} / \\mathrm{sec}$ and $5 \\mathrm{~m} / \\mathrm{sec}$. At what time will the distance between the two points be $26 \\mathrm{~m}$. (Without using trigonometry!)",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173382,
        "difficulty": null
      },
      {
        "id": "d41a02065a53c0ebbceabdeef48ebf7d1193ddaab46cb4cf0b2440b518f50a56",
        "domain": "numina",
        "question": "A square-based pyramid has every edge 1 unit long. What is the radius of the sphere inscribed in the pyramid?",
        "answer": "\\frac{\\sqrt{6}-\\sqrt{2}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170323,
        "difficulty": null
      },
      {
        "id": "4742446b581bb6df769fbfbd93f93e47e821838471b00623410963a1210e51c9",
        "domain": "numina",
        "question": "We roll a die until a six comes up. What is the probability that we do not roll a five in the meantime?",
        "answer": "\\frac{1}{2}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 166326,
        "difficulty": null
      },
      {
        "id": "ccfea27aaa56d4575d0d6629e20136d2e8000cd7d308f3dcd66c702d6e1a42ac",
        "domain": "numina",
        "question": "Lisa considers the number\n\n$$\nx=\\frac{1}{1^{1}}+\\frac{1}{2^{2}}+\\cdots+\\frac{1}{100^{100}} .\n$$\n\nLisa wants to know what $x$ is when rounded to the nearest integer. Help her determine its value.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90762,
        "difficulty": null
      },
      {
        "id": "49be69e538854519d639ecdfe223e6556438a2fcc23a27357e076b71d940f580",
        "domain": "numina",
        "question": "We throw five dice of different colors. In how many ways can the sum of the throws be 11?",
        "answer": "205",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 167439,
        "difficulty": null
      },
      {
        "id": "1afe4030a21bf662c5ee08a52fff290ba6cbca6c450d849e495901d6db131c57",
        "domain": "numina",
        "question": "Task B-3.7. The length of the side of an equilateral triangle is $6 \\mathrm{~cm}$. A line parallel to one of the sides has been drawn through its centroid. Calculate the volume of a right prism whose base is the resulting trapezoid, and the length of the prism's height is equal to the height of the trapezoid. Determine the cosine of the angle that the spatial diagonal of the prism forms with the base plane.",
        "answer": "\\sqrt{\\frac{28}{31}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89219,
        "difficulty": null
      },
      {
        "id": "742ca3c640b8821f3494f14d64674b49f2a337a46524c92e5347a07d0518b1fe",
        "domain": "numina",
        "question": "Example 1. Calculate the definite integral $\\int_{1}^{4} x^{2} d x$.",
        "answer": "21",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32742,
        "difficulty": null
      },
      {
        "id": "86b42d94f1026a1f0fb44e35fc9d12777f6a88226d6514aa874651cfc1f28ada",
        "domain": "numina",
        "question": "Second Part (8 points per question, total 40 points)\n6. A box contains 10 black balls, 9 white balls, and 8 red balls. If you close your eyes and draw balls from the box, how many balls must you draw to ensure that you have at least one red ball and one white ball? $\\qquad$",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119953,
        "difficulty": null
      },
      {
        "id": "9b919fc547d0102e062f16ad39cdb606c7c005e705073f767e6ef0e1706ec582",
        "domain": "numina",
        "question": "Problem 11.4. On a horizontal floor, there are three volleyball balls with a radius of 18, each touching the other two. Above them, a tennis ball with a radius of 6 is placed, touching all three volleyball balls. Find the distance from the top point of the tennis ball to the floor. (All balls are spherical.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_9703912434d201fc4f33g-03.jpg?height=385&width=521&top_left_y=84&top_left_x=466)",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12181,
        "difficulty": null
      },
      {
        "id": "b9cee9d8ec92ea6bad9d9edaf83bbea80a0882410b8d20297757781b35e2a69a",
        "domain": "numina",
        "question": "11. There are 11 students who have signed up for the volunteer tour guide activity at the museum. The activity runs from 9 AM to 5 PM, with a public mini-lecture every hour. Each session requires 1 student to provide tour guide services to visitors. To avoid overworking the students, the museum will not schedule the same person for two consecutive sessions, and the first and last sessions must be handled by two different students. How many scheduling arrangements are possible for the museum?",
        "answer": "100000010",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 127206,
        "difficulty": null
      },
      {
        "id": "ac4d5d76b733b7e8f2468ee1234fdd565356aef36d9cb22e0f15074d6210f35b",
        "domain": "numina",
        "question": "8. (3 points) In $17 \\square 17 \\square 17 \\square 17$, fill in the four $\\square$ with the operation symbols “+”, “-”, “$\\times$”, “$\\div$” each once. The maximum value of the resulting expression is $\\qquad$ .",
        "answer": "305",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120520,
        "difficulty": null
      },
      {
        "id": "02276ef16f69ff82709be68611c9603997f023ad3c0da429f85c1a0f169e1835",
        "domain": "numina",
        "question": "Let's determine the greatest common divisor of the numbers $A$ and $C$, as well as $B$ and $C$.\n\n$$\n\\begin{aligned}\n& A=177^{5}+30621 \\cdot 173^{3}-173^{5} \\\\\n& B=173^{5}+30621 \\cdot 177^{3}-177^{5} \\\\\n& C=173^{4}+30621^{2}+177^{4}\n\\end{aligned}\n$$",
        "answer": "30637",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169146,
        "difficulty": null
      },
      {
        "id": "535712c367c94b211b4d425bddfb23af8cffcd955525d5b00388d4ef3687ea40",
        "domain": "numina",
        "question": "7. A survey was conducted involving 420 families. It was found that $95\\%$ of the families own a TV, and $80\\%$ of the families own a computer. A family is randomly selected. If it is known that each surveyed family has at least one of the mentioned devices, what is the probability that the selected family owns a computer but not a TV? Express the probability as a percentage.\n\nNo calculators or reference materials are allowed.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 87090,
        "difficulty": null
      },
      {
        "id": "82afba3d0e0210ac870974690eb35de5a4ba2d1942ad45c24823bc6a741922c2",
        "domain": "numina",
        "question": "A box of chocolates in the shape of a cuboid was full of chocolates arranged in rows and columns. Misha ate some of them and rearranged the remaining ones so that they filled (without gaps) up to one place three whole rows. The remaining chocolates from the next incomplete row were eaten. Then the remaining chocolates were rearranged and filled five columns without one place. The chocolates in the incomplete column were eaten again. Thus, only a third of the original number of chocolates remained in the box. Find:\n\na) how many chocolates the entire box had,\n\nb) how many chocolates Misha ate before he did the first rearrangement.\n\n(Bednářrová)",
        "answer": "25",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76236,
        "difficulty": null
      },
      {
        "id": "f345a7e31db190f2d94cb2451d1659e4d2f3f593e80630500b38d249cff78022",
        "domain": "numina",
        "question": "21. Each of the integers $1,2,3, \\ldots, 9$ is assigned to each vertex of a regular 9 -sided polygon (that is, every vertex receives exactly one integer from $\\{1,2, \\ldots, 9\\}$, and two vertices receive different integers) so that the sum of the integers assigned to any three consecutive vertices does not exceed some positive integer $n$. What is the least possible value of $n$ for which this assignment can be done?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154501,
        "difficulty": null
      },
      {
        "id": "a10433a987e81eae2c7581e0f0f9ea9d466122d177d74e25776abd585385fbf1",
        "domain": "numina",
        "question": "5. During the shooting practice, each soldier fired 10 times. One of them completed the task successfully and scored 90 points. How many times did he score 7 points, if he hit the bullseye 4 times, and the results of the other hits were sevens, eights, and nines? There were no misses at all.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10310,
        "difficulty": null
      },
      {
        "id": "32b34835bf078ddbb09180d704747ccb00dc3abcd40e020dc031ef3a1c4b94d6",
        "domain": "numina",
        "question": "8. (5 points) If two 0s are added after a number, the new number is 1980 more than the original number. What is this number? $\\qquad$",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98119,
        "difficulty": null
      },
      {
        "id": "e33c7c9907c7f98d1e17a4049e159845b291d065ef6d3e1a68561cb098ad52db",
        "domain": "numina",
        "question": "Test $\\mathbf{G}$ Calculation:\n$$\n\\frac{\\left(2^{4}+\\frac{1}{4}\\right)\\left(4^{4}+\\frac{1}{4}\\right)\\left(6^{4}+\\frac{1}{4}\\right)\\left(8^{4}+\\frac{1}{4}\\right)\\left(10^{4}+\\frac{1}{4}\\right)}{\\left(1^{4}+\\frac{1}{4}\\right)\\left(3^{4}+\\frac{1}{4}\\right)\\left(5^{4}+\\frac{1}{4}\\right)\\left(7^{4}+\\frac{1}{4}\\right)\\left(9^{4}+\\frac{1}{4}\\right)} .\n$$\n(1991 Jiangsu Province Junior High School Mathematics Competition Question)",
        "answer": "221",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136811,
        "difficulty": null
      },
      {
        "id": "825bfaa1cdb47931f6b0d6a3fed1c3dc91223ee0371558a10a74386b6cd0efa5",
        "domain": "numina",
        "question": "2. Three candles were lit simultaneously. When the first candle burned out, $\\frac{2}{5}$ of the second candle remained, and $\\frac{3}{7}$ of the third candle remained. What part of the third candle will remain when the second candle burns out?",
        "answer": "\\frac{1}{21}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13030,
        "difficulty": null
      },
      {
        "id": "e51daaa185d85b35ee7b58a18946a965741bf8dff3ea85ced1aed8286f2e0f75",
        "domain": "numina",
        "question": "## Task B-2.7.\n\nA line passing through the vertex $C$ of the right angle of triangle $A B C$ with the shorter leg forms an angle of $30^{\\circ}$ and divides the hypotenuse in the ratio 1 : 2. If the length of the shorter leg is $\\sqrt{3}$, determine the length of the hypotenuse.",
        "answer": "\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88233,
        "difficulty": null
      },
      {
        "id": "19f6b6e45a06f73df9cf72a8d457a7409660d3e753c76e81900df9b9d3c63f74",
        "domain": "numina",
        "question": "6- 16 Find the value of $10 \\operatorname{ctg}(\\operatorname{arcctg} 3+\\operatorname{arcctg} 7+\\operatorname{arcctg} 13+\\operatorname{arcctg} 21)$.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125403,
        "difficulty": null
      },
      {
        "id": "4c675e2d94573390a61259b86d94b54cda99aeb63b547e68c43c92c63c74b425",
        "domain": "numina",
        "question": "3. Given real numbers $a, b, c$ satisfy\n$$\nf(x)=a \\cos x+b \\cos 2 x+c \\cos 3 x \\geqslant-1\n$$\n\nfor any real number $x$. Then the maximum value of $a+b+c$ is . $\\qquad$",
        "answer": "3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 122912,
        "difficulty": null
      },
      {
        "id": "bd96dca97e6df59e1d27c32256706f435f0016da915dee3d87acbe7133df445e",
        "domain": "numina",
        "question": "Task 2. (10 points) A circle with a radius of 10 touches two adjacent sides $AB$ and $AD$ of the square $ABCD$. On the other two sides, the circle intersects at points, cutting off segments of 4 cm and 2 cm from the vertices, respectively. Find the length of the segment that the circle cuts off from vertex $B$ at the point of tangency.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7290,
        "difficulty": null
      },
      {
        "id": "77c6280ad9d26ac22dab5d7372b93b7b95e2fe393e70994df4aabf54f193a21e",
        "domain": "numina",
        "question": "Problem 6. (Option 1).\n\nIn the right triangle $ABC: \\angle ACB=90^{\\circ}, AC=6, BC=4$. On the line $BC$, a point $D (CD>BD)$ is marked such that $\\angle ADC=45^{\\circ}$. On the line $AD$, a point $E$ is marked such that the perimeter of triangle $CBE$ is the smallest possible. Then, on the line $DC$, a point $F$ is marked such that the perimeter of triangle $AFE$ is the smallest possible. Find $CF$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_3fdcc7ef9dfdd3dab8ecg-07.jpg?height=846&width=557&top_left_y=228&top_left_x=241)\n\n## Construction and Proof:\n\nFor convenience, construct a square ACDK (AD is its diagonal, since $\\angle ADC=45^{\\circ}$. Mark point $B_1$ on side $DK (B_1D=BD \\Rightarrow B_1B \\perp AD)$.\n\nDraw line $B_1C$, which intersects $AD$ at point $E$.\n\nThe perimeter of triangle $CBE$ is the smallest because among all possible points $E_1$ on line $AD$, the sum of the lengths of segments $B_1E + EC$ is the smallest ($B_1E + EC < B_1E_1 + E_1C$ - triangle inequality) and $B_1E = EB$. Similarly, mark point $A_1$ on side $AC$ ($A_1C = AC$). Draw line $A_1E$, which intersects $CD$ at point $F$.\n\nThe perimeter of triangle $AFE$ is the smallest because among all possible points $F_1$ on line $AD$, the sum of the lengths of segments $A_1F + EF$ is the smallest ($A_1F + EF < A_1F_1 + F_1E$ - triangle inequality) and $A_1F = FA$.",
        "answer": "3.6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10429,
        "difficulty": null
      },
      {
        "id": "9b3ebddd83bd8fdd24776a4e1022110af6c810a902e4e675c37f127f6d458567",
        "domain": "numina",
        "question": "【Question 12】A and B take turns marking numbers from 1 to 17, with the rules: you cannot mark a number that has already been marked; you cannot mark a number that is twice a marked number; you cannot mark a number that is half of a marked number; the player who cannot mark a number loses. Now A marks 8 first, to ensure a certain win, B should mark _ next. $\\qquad$",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 102288,
        "difficulty": null
      },
      {
        "id": "d2416888a400045c270b5ee5bb1ab28f23ddf7e5f098cd60710b8245767b1d30",
        "domain": "numina",
        "question": "Let $a, b, c, d$ be real numbers defined by\n\n$$\na=\\sqrt{4-\\sqrt{5-a}}, b=\\sqrt{4+\\sqrt{5-b}}, c=\\sqrt{4-\\sqrt{5+c}}, d=\\sqrt{4+\\sqrt{5+d}}\n$$\n\nCalculate their product.\n\n## 2 Solutions",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 80972,
        "difficulty": null
      },
      {
        "id": "93f69bc6172bca4edc55139bd9c1439021a421dbbc78480edd739dfad8f892ce",
        "domain": "numina",
        "question": "10.4 Find the smallest positive integer $k$, for which in any coloring of the numbers of the set $M=\\{1,2,3, \\ldots, k\\}$ in two colors, there will be ten not necessarily distinct numbers of the same color from the set $M$",
        "answer": "109",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13426,
        "difficulty": null
      },
      {
        "id": "5b62c204405fc3dc068d44122d54668ef4000e31c81bf9fd177f988c79feb759",
        "domain": "numina",
        "question": "Two workers are working on identical workpieces and both are currently meeting the standard. With the application of an innovation, one completes a workpiece 5 minutes faster than the standard time, while the other completes it 3 minutes faster. As a result, their average performance increases to 137.5%. What is the standard time for completing one workpiece?",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 174206,
        "difficulty": null
      },
      {
        "id": "60085a9f0acd836175f45ca90e2e43e8f967ab3931945f7bf07cab1e42c182a4",
        "domain": "numina",
        "question": "4. What is the minimum number of cells that need to be colored on a $6 \\times 6$ board so that, for any placement (including rotations and flips) of a figure consisting of 4 cells in the shape of the letter $\\Gamma$, there is at least one colored cell?",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 15560,
        "difficulty": null
      },
      {
        "id": "060afa2b843e84e67ea321a5bb2ac52b0e33f2d167131746fd7e1cf380eea35a",
        "domain": "numina",
        "question": "For real numbers $a$ and $b$, where $|a| \\neq|b|$ and $a \\neq 0$, the following holds:\n\n$$\n\\frac{a-b}{a^{2}+a b}+\\frac{a+b}{a^{2}-a b}=\\frac{3 a-b}{a^{2}-b^{2}}\n$$\n\nDetermine the value of the expression $\\frac{b}{a}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162778,
        "difficulty": null
      },
      {
        "id": "fde5cf3eefb652824e2d4214533893ba38775e509ef3883366b8c09c78963120",
        "domain": "numina",
        "question": "H2. How many nine-digit integers of the form 'pqrpqrpqr' are multiples of 24 ? (Note that $p, q$ and $r$ need not be different.)",
        "answer": "112",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 155246,
        "difficulty": null
      },
      {
        "id": "186a2041c6aa078759428354e4cfeb129ddc90b8daeb49073d5043756a7d28f0",
        "domain": "numina",
        "question": "Extend 1 Let non-negative real numbers $a, b, c, x, y, z$ satisfy $a+b+c=x+y+z=1$.\nFind the minimum value of $\\left(a-x^{2}\\right)\\left(b-y^{2}\\right)\\left(c-z^{2}\\right)$.",
        "answer": "-\\frac{1}{4}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 136719,
        "difficulty": null
      },
      {
        "id": "2f9c3a3b84652ae670e8495dcddc5a68cb952ca314b2c38f1549dcdfb529de7d",
        "domain": "numina",
        "question": "B2. Integers $a, b, c, d$, and $e$ satisfy the following three properties:\n(i) $2 \\leq a<b<c<d<e<100$\n(ii) $\\operatorname{gcd}(a, e)=1$\n(iii) $a, b, c, d, e$ form a geometric sequence.\nWhat is the value of $c$ ?",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 106059,
        "difficulty": null
      },
      {
        "id": "1b91910e8ad060b05686715b7061c2aea4b067cc36d822affc42e9cd61c4bcd5",
        "domain": "numina",
        "question": "3 3.7 A regular dodecahedron has 20 vertices and 30 edges. Each vertex is the intersection of 3 edges, and the other endpoints of these 3 edges are 3 other vertices of the regular dodecahedron. We call these 3 vertices adjacent to the first vertex. Place a real number at each vertex, such that the number placed at each vertex is exactly the arithmetic mean of the numbers placed at the 3 adjacent vertices. Let $M, m$ be the largest and smallest of these 20 numbers, respectively. What is the range of $M-m$?",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125671,
        "difficulty": null
      },
      {
        "id": "98de1ad12e46856b17ace80571eb73b57acf342237580a9d1a7a57853b9d70dd",
        "domain": "numina",
        "question": "I1.4 In the coordinate plane, the distance from the point $(-C, 0)$ to the straight line $y=x$ is $\\sqrt{D}$, find the value of $D$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153140,
        "difficulty": null
      },
      {
        "id": "1bba79b5cf14e4d1095c84ec7c2249b56b0cd917821e1e73ddc332461ac17322",
        "domain": "numina",
        "question": "33rd BMO 1997 Problem 3 Find the number of polynomials of degree 5 with distinct coefficients from the set {1, 2, ... , 9} which are divisible by x 2 - x + 1. Solution",
        "answer": "624",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 149226,
        "difficulty": null
      },
      {
        "id": "4a7687c3ed1b8c78f8a369473ad66af808a1a229ce89af746f7e0ebaf8bcf26e",
        "domain": "numina",
        "question": "5. Given a natural number $x=6^{n}+1$, where $n-$ is an odd natural number. It is known that $x$ has exactly three distinct prime divisors, one of which is 11. Find $x$.",
        "answer": "7777",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 7834,
        "difficulty": null
      },
      {
        "id": "854e4c32a74f2f92221bc8db0b706b58fb1a0e595fdcbbfbb2b197fa47d27343",
        "domain": "numina",
        "question": "14. (10 points) Using a batch of paper to bind a type of exercise book. If 120 books have already been bound, the remaining paper is $40 \\%$ of the total; if 185 books have been bound, then 1350 sheets of paper are left. How many sheets of paper are there in total?",
        "answer": "18000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122362,
        "difficulty": null
      },
      {
        "id": "ae7aa0df541d9b1c573c1c2f801d6578c47d12cbde5708c53b09011d10cc5828",
        "domain": "numina",
        "question": "1. Let $f(x)=x^{3}+a x+b$, with $a \\neq b$, and suppose the tangent lines to the graph of $f$ at $x=a$ and $x=b$ are parallel. Find $f(1)$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152296,
        "difficulty": null
      },
      {
        "id": "d7173b08ab5af48d899b6b50192675ef4af683cfddb62bc46318b819cce141ce",
        "domain": "numina",
        "question": "Problem 9.5. Let $A B C D$ be a cyclic quadrilateral with circumcircle $k$. The rays $\\overrightarrow{D A}$ and $\\overrightarrow{C B}$ meet at point $N$ and the line $N T$ is tangent to $k, T \\in k$. The diagonals $A C$ and $B D$ meet at the centroid $P$ of $\\triangle N T D$. Find the ratio $N T: A P$.\n\nIvailo Kortezov",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 177627,
        "difficulty": null
      },
      {
        "id": "3c7047f6a275118f75a40556abca439278b78e9fc620d8d1775a94e6d3dd4c9a",
        "domain": "numina",
        "question": "## problem statement\n\nApproximately calculate using the differential.\n\n$y=\\sqrt{x^{2}+5}, x=1.97$",
        "answer": "2.98",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46696,
        "difficulty": null
      },
      {
        "id": "baa84920d7a068ee923e2dbd7a024d8c3c01f9c5055e4bf1ff6454842a0b0d79",
        "domain": "numina",
        "question": "Find the remainder when $10^{2^{0}}+10^{2^{1}}+\\cdots+10^{2^{2021}}$ is divided by $\\underbrace{44 \\cdots 44}_{44}$.",
        "answer": "2020202020202020202020202020202020202020110",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 90206,
        "difficulty": null
      },
      {
        "id": "692726033245451c0ffddc5afb5864e46c17363b67d4f7a1e4bd2a6a32f58a40",
        "domain": "numina",
        "question": "35. Let $x$ be a positive integer, and write $a=\\left\\lfloor\\log _{10} x\\right\\rfloor$ and $b=\\left\\lfloor\\log _{10} \\frac{100}{x}\\right\\rfloor$. Here $\\lfloor c\\rfloor$ denotes the greatest integer less than or equal to $c$. Find the largest possible value of $2 a^{2}-3 b^{2}$.",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160510,
        "difficulty": null
      },
      {
        "id": "f5fc83b9611128d8036d6df6e8d1e8f01d9d370d9a954a6b8815b21d8b86dadb",
        "domain": "numina",
        "question": "6. In tetrahedron $ABCD$, $AD=2\\sqrt{3}$, $\\angle BAC=60^{\\circ}$, $\\angle BAD=\\angle CAD=45^{\\circ}$. If a sphere passing through point $D$ is tangent to plane $ABC$ and internally tangent to the circumscribed sphere of the tetrahedron with a radius of 1, then the radius of the circumscribed sphere of tetrahedron $ABCD$ is . $\\qquad$",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 128875,
        "difficulty": null
      },
      {
        "id": "ec1ee5de183dc20ce26816feede09a633cc59b2c94bda978733f1954c7883b17",
        "domain": "numina",
        "question": "3. $f(x)$ is an odd function, $g(x)$ is an even function, and $f(x)+g(x)+f(x) g(x)=\\left(e^{x}+\\sin x\\right) \\cdot \\cos x$, then $f(x)$ has $\\qquad$ zeros in $[0,4 \\pi)$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135407,
        "difficulty": null
      },
      {
        "id": "ab5a374005945b3406ab909608765ba0ef7d56dd65529af351d09c0ca02cac8e",
        "domain": "numina",
        "question": "Example 3. Find $\\lim _{x \\rightarrow \\frac{\\pi}{2}} \\frac{\\tan x}{\\tan 3 x}$.",
        "answer": "3",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32644,
        "difficulty": null
      },
      {
        "id": "7f19ddb1410314b50aecff500a215d9541d844f2888bd2bea631eee08050b969",
        "domain": "numina",
        "question": "2.2. Gavriila found out that the front tires of the car last for 24000 km, while the rear tires last for 36000 km. Therefore, he decided to swap them at some point to maximize the distance the car can travel. Find this distance (in km).",
        "answer": "28800",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11178,
        "difficulty": null
      },
      {
        "id": "f96651afccd1643e51c802d67f0c2750126d5f12af30fa1918d20d60f879f72e",
        "domain": "numina",
        "question": "1. (12th \"Hope Cup\" Invitational Training Question) In the quadratic trinomial $a x^{2}+b x+c$, $a>100$, how many integer values of $x$ at most can make the absolute value of the quadratic trinomial not exceed 50?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122570,
        "difficulty": null
      },
      {
        "id": "32120f8635a3c7b403e9022c2c417e7b25b1bd9a17226436871b65d043258e4c",
        "domain": "numina",
        "question": "7.6. Let $a, b, c$ be natural numbers, and the product $ab$ is divisible by $5c$, the product $c$ is divisible by $13a$, and the product $ca$ is divisible by $31b$. Find the smallest possible value of the product $abc$. Justify your answer.",
        "answer": "4060225",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17434,
        "difficulty": null
      },
      {
        "id": "29ffb0ef6eae154ad7e3a5ae2fc6facec115e71c52bc78f69e4367642682b7fb",
        "domain": "numina",
        "question": "Example 2. Find the smallest positive integer $n$, such that in any 9-vertex and $n$-edge graph with edges colored in two colors, there must exist a monochromatic triangle.\n\n untranslated portion: \n(There is no additional text to translate, as the provided text has been fully translated.)",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96227,
        "difficulty": null
      },
      {
        "id": "a4777b3d5e917a522e5a6b3e8b1b4dcdf92e27b207f0e606f8bb0e03b8e20ab3",
        "domain": "numina",
        "question": "2. Find the smallest natural number that has exactly 12 different natural divisors, the largest prime divisor of which is the number 101, and the last digit is zero.",
        "answer": "2020",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14583,
        "difficulty": null
      },
      {
        "id": "c68a11070b9754b199ef3bcb3e9a9ffc1d5d0173bf1081bdcd5f5b35aab21ce5",
        "domain": "numina",
        "question": "10. (20 points) Given $O$ as the origin, $F$ as the right focus of the ellipse $C: \\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$, a line $l$ passing through point $F$ intersects the ellipse $C$ at points $A$ and $B$. Points $P$ and $Q$ on the ellipse satisfy\n$$\n\\overrightarrow{O P}+\\overrightarrow{O A}+\\overrightarrow{O B}=\\overrightarrow{O P}+\\overrightarrow{O Q}=0 \\text {, }\n$$\n\nand points $P, A, Q, B$ are concyclic. Find the eccentricity of the ellipse $C$.",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 136173,
        "difficulty": null
      },
      {
        "id": "3a77b1b1d83c1fbb6899645b3ac21ad284b110a01ef14a0b07180c256ef074dd",
        "domain": "numina",
        "question": "$$\n[\\sqrt{2-\\sqrt{3}}(2-\\sqrt{3})+\\sqrt{2+\\sqrt{3}}(2+\\sqrt{3})]^{2}=?\n$$",
        "answer": "54",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 174956,
        "difficulty": null
      },
      {
        "id": "7c0a679417f0bdb94a4a809bdb3000708f11eb3c5405fc0a11707a0a927a252e",
        "domain": "numina",
        "question": "97. The Fibonacci sequence. Consider the Fibonacci sequence $1,1,2,3,5,13, \\ldots$, whose terms $F_{n}$ (Fibonacci numbers) satisfy the relation $F_{n+2}=F_{n}+F_{n+1}$, $F_{1}=F_{2}=1$. Now consider the sequence of digits in the units place of the Fibonacci numbers. Will this sequence be cyclic, that is, can it be obtained by the unlimited repetition of the same finite set of digits, as is true, for example, in the case of the sequence $055055 \\ldots$.",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41530,
        "difficulty": null
      },
      {
        "id": "fa1165e829efc2d6084b85a92ac55a3e915648c649c822bcec764572100098c9",
        "domain": "numina",
        "question": "Task 1. 15 points\n\nFind the minimum value of the expression\n\n$$\n\\sqrt{x^{2}-\\sqrt{3} \\cdot|x|+1}+\\sqrt{x^{2}+\\sqrt{3} \\cdot|x|+3}\n$$\n\nas well as the values of $x$ at which it is achieved.",
        "answer": "\\sqrt{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7545,
        "difficulty": null
      },
      {
        "id": "858de3afd5e2c347a55e786d58050d97d285a464bfd4c03d474661eeff2117bd",
        "domain": "numina",
        "question": "$\\underline{\\text { Folklore }}$\n\nFind the largest natural $n$, for which $n^{200}<5^{300}$.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68991,
        "difficulty": null
      },
      {
        "id": "088815b1bc67618bdd491953b731faad06e72451d06b1b846fcb7ff80a0d7d54",
        "domain": "numina",
        "question": "1. Calculate: $25 \\times 13 \\times 2 + 15 \\times 13 \\times 7=$",
        "answer": "2015",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119879,
        "difficulty": null
      },
      {
        "id": "8f92a8ab19892b9344a2ab8abb48005cd56b5f88c3e348a57fd5332dad4d6b7d",
        "domain": "numina",
        "question": "1. (5 points) If $0.142857^{+} x=1.5$, then $x=$",
        "answer": "1\\frac{5}{14}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140946,
        "difficulty": null
      },
      {
        "id": "99f57475387a0f3ca4eb560b3b7fff6cc13af88a28b7f4d01c272e80c02772d7",
        "domain": "numina",
        "question": "11. Real numbers $x, y, z, w$ satisfy $x+y+z+w=1$, then the maximum value of $M=x w+2 y w+3 x y+3 z w+4 x z+5 y z$ is $\\qquad$ .",
        "answer": "\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140650,
        "difficulty": null
      },
      {
        "id": "e2e616945540bcdd10b68c8a8c64065bd8ca1c4ca059cbd7c7856e77fe85f4ba",
        "domain": "numina",
        "question": "19. In the triangle $A B C, A B=A C, \\angle A=90^{\\circ}, D$ is the midpoint of $B C, E$ is the midpoint of $A C$ and $F$ is a point on $A B$ such that $B E$ intersects $C F$ at $P$ and $B, D, P, F$ lie on a circle. Let $A D$ intersect $C P$ at $H$. Given $A P=\\sqrt{5}+2$, find the length of $P H$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160973,
        "difficulty": null
      },
      {
        "id": "f92ba6c94d0d9db9484759935597b7ce6b1adfe32a19bc7817e5084edea3e9ed",
        "domain": "numina",
        "question": "47 (1289). Two brothers walk together from school to home at the same speed. One day, 15 minutes after leaving school, the first brother ran back to school and, upon reaching it, immediately set off to catch up with the second. Left alone, the second continued walking home at half his usual speed. When the first brother caught up with the second, they resumed their initial speed and arrived home 6 minutes later than usual. How many times faster is the first brother's running speed compared to their usual walking speed?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35919,
        "difficulty": null
      },
      {
        "id": "57e70a72ea37983747b7fb2546fbe7f04ac682603e81db110cba72b95fda8c81",
        "domain": "numina",
        "question": "18. (6 points) Given the four-digit number $\\overline{\\mathrm{ABCD}}$, the conclusions of three people, Jia, Yi, and Bing, are as follows:\n\nJia: “The unit digit is half of the hundred digit”;\nYi: “The tens digit is 1.5 times the hundred digit”;\nBing: “The average of the four digits is 4”.\nAccording to the above information, we can get: $\\overline{\\mathrm{ABCD}}=$",
        "answer": "4462",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 101132,
        "difficulty": null
      },
      {
        "id": "2c67853f12c5e0ce1db5151842c6175a49c08cd4835e79ee30f24c4a6fa5d929",
        "domain": "numina",
        "question": "6. A line with slope $k$ is drawn through one of the foci of the ellipse $x^{2}+2 y^{2}=3$, intersecting the ellipse at points $A, B$. If $|A B|=2$, then $|k|=$ $\\qquad$ .",
        "answer": "\\sqrt{1+\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 103330,
        "difficulty": null
      },
      {
        "id": "d9268cee7e755f6dc0bbc3d134980443cc16e6baf93dc970c0225e0c18aa7c0d",
        "domain": "numina",
        "question": "Exercise 4. We are given 20 points in the plane such that no three of them are collinear. How many lines can be formed passing through two of these points?",
        "answer": "190",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 78158,
        "difficulty": null
      },
      {
        "id": "668261a37d1d11ee7fc3a518899987a82b956f0f7433c3e5ce3be218f051c3e8",
        "domain": "numina",
        "question": "(11) As shown in the figure, let $S-ABCD$ be a pyramid with a height of 3, where the base $ABCD$ is a square with side length 2, and the projection of vertex $S$ onto the base is the center of the square $ABCD$. $K$ is the midpoint of edge $SC$, and a plane through $AK$ intersects segments $SB$ and $SD$ at points $M$ and $N$ respectively. ($M$ and $N$ can be the endpoints of the segments).\n(1) Find the sine of the angle between line $AK$ and plane $SBC$;\n(2) Prove that $\\frac{V_{S-MKN}}{V_{S-BCD}}=\\frac{SM \\cdot SK \\cdot SN}{SB \\cdot SC \\cdot SD}$, and find the volume of the pyramid $S-AMKN$ when $M$ is the midpoint of $SB$.",
        "answer": "\\frac{3}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 106825,
        "difficulty": null
      },
      {
        "id": "a366e4caaf9d26a95789667994339995db440322323ae67f7a25801d747ead64",
        "domain": "numina",
        "question": "11. The selling price of a commodity is an integer number of yuan. 100 yuan can buy at most 3 pieces. Person A and Person B each brought several hundred-yuan bills. The money A brought can buy at most 7 pieces of this commodity, and the money B brought can buy at most 14 pieces. When the two people's money is combined, they can buy 1 more piece. The price of each piece of this commodity is yuan.",
        "answer": "27",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112077,
        "difficulty": null
      },
      {
        "id": "5ea3ddfd4d48c6f467b64345f58799e1a05eca1b17f054fd94d70e73baaf9cad",
        "domain": "numina",
        "question": "Question 42: A set of 4027 points on a plane is called a \"Colombian point set\" if no three points are collinear, and 2013 points are red, 2014 points are blue. A set of lines can divide the plane into several regions. If a set of lines for a Colombian point set satisfies the following two conditions, it is called a \"good set of lines\":\n(1) These lines do not pass through any of the points in the Colombian point set;\n(2) No region contains points of both colors.\nFind the minimum value of $\\mathrm{k}$ such that for any Colombian point set, there exists a good set of lines consisting of $\\mathrm{k}$ lines.",
        "answer": "2013",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 95029,
        "difficulty": null
      },
      {
        "id": "41564787f9e93580385d8cef08f6d0e66227eb644686602618c0d318c0281b04",
        "domain": "numina",
        "question": "## Task 2\n\nA Pioneer group goes to the castle pond in Karl-Marx-City to go boating. 23 Pioneers get into the boats. The other 5 Pioneers wait on the shore and play with a ball.\n\nHow many Pioneers are in the group?",
        "answer": "28",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 186598,
        "difficulty": null
      },
      {
        "id": "23f844c175fff35ea05397dda19fa62a7903156720e8b870ff24381457fb976b",
        "domain": "numina",
        "question": "12. What is the value of $A^{2}+B^{3}+C^{5}$, given that:\n$$\n\\begin{array}{l}\nA=\\sqrt[3]{16 \\sqrt{2}} \\\\\nB=\\sqrt{9 \\sqrt[3]{9}} \\\\\nC=\\left[(\\sqrt[5]{2})^{2}\\right]^{2}\n\\end{array}\n$$",
        "answer": "105",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156482,
        "difficulty": null
      },
      {
        "id": "d6fcde5d16abff4e58426dc8e96066184bbea763311937f98e09c66fca4e7505",
        "domain": "numina",
        "question": "11.1. Find all positive integer solutions of the equation $(n+2)!-(n+1)!-(n)!=n^{2}+n^{4}$. Answer. $n=3$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9787,
        "difficulty": null
      },
      {
        "id": "ea16034f7ccd35c97258b2d6cdd36c91688c4595b950ccd530e71f45bf82832a",
        "domain": "numina",
        "question": "For the vertices of an $n$-sided polygon, we wrote different real numbers such that any written number is the product of the numbers written at the two adjacent vertices.\n\nDetermine the value of $n$!",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 168970,
        "difficulty": null
      },
      {
        "id": "0d80d04163bc77c600ecf25f4c0f4e5a81882b389bc88977213eb5db8f6925b0",
        "domain": "numina",
        "question": "Which is the three-digit (integer) number that, when increased or decreased by the sum of its digits, results in a number consisting of the same digit repeated?",
        "answer": "105",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 175539,
        "difficulty": null
      },
      {
        "id": "4e37780ac79f78ffac2c6ae3ad0de0f0aef66026e5248b280125397166f504dd",
        "domain": "numina",
        "question": "Example 1 Calculate the sum $S=\\sum_{n=1}^{509}\\left[\\frac{511 n}{509}\\right]$.",
        "answer": "130051",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94066,
        "difficulty": null
      },
      {
        "id": "6641b88f138dceb15e5efdf632d721245b5016a491342fb35339b956babb49f9",
        "domain": "numina",
        "question": "3. Pasha and Igor are flipping a coin. If it lands on heads, Pasha wins; if tails, Igor wins. The first time the loser pays the winner 1 ruble, the second time - 2 rubles, then - 4, and so on (each time the loser pays twice as much as the previous time). At the beginning of the game, Pasha had a single-digit amount of money, and Igor had a four-digit amount. By the end, Igor had a two-digit amount, and Pasha had a three-digit amount. What is the minimum number of games Pasha could have won? The players cannot go into debt.\n\n(L. S. Korechkova, A. A. Tessler)",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 11801,
        "difficulty": null
      },
      {
        "id": "052c803921923652d853d4bf883be97591bc6bc58ef104ec052b9f0c1a5f7c23",
        "domain": "numina",
        "question": "$1 \\cdot 30$ Find the largest natural number $x$, such that for every natural number $y, x$ divides\n$$\n7^{y}+12 y-1 \\text {. }\n$$\n(\"Friendship Cup\" International Mathematics Competition, 1992)",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 92771,
        "difficulty": null
      },
      {
        "id": "4cfce894ba272f26c5ca79dd3e3f14a6017fd5aa0ad80d7620d03a5ce093fc42",
        "domain": "numina",
        "question": "4. On a plane, there are 10 different points. We considered the midpoints of all segments connecting all pairs of points. What is the smallest number of midpoints that could have arisen?",
        "answer": "17",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16159,
        "difficulty": null
      },
      {
        "id": "1719ee245508bce57128080d42ebf15ddb744dde38d285540e634d57e4ce6de3",
        "domain": "numina",
        "question": "1. [3] How many different values can $\\angle A B C$ take, where $A, B, C$ are distinct vertices of a cube?",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150947,
        "difficulty": null
      },
      {
        "id": "17ab771e9fd72086e455fb6efb4a71ff2fb5c6bb3fe2d96557fec7f20310af4c",
        "domain": "numina",
        "question": "## Task 1\n\nParents of class 2a want to host guests for the festival. 8 parents each host 3 guests, 7 parents each host 2 guests, and 9 parents each host 1 guest.\n\nHow many guests will be hosted by the parents of class 2a?",
        "answer": "47",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186482,
        "difficulty": null
      },
      {
        "id": "1deb23d9091e09156a2931eab5405d180d50e5d84c8e73ee29849a69d0befc91",
        "domain": "numina",
        "question": "6. (10 points) The sequence $1,1,2,3,5,8, \\cdots$ is such that each term from the second onward is equal to the sum of the two preceding terms, forming the Fibonacci sequence. Each term in this sequence is called a Fibonacci number. It can be proven that \"any positive integer $n$ can be expressed as the sum of several different Fibonacci numbers.\" How many ways are there to express 100 as the sum of several different Fibonacci numbers? (Different orders of the addends are considered the same method)",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 106607,
        "difficulty": null
      },
      {
        "id": "506d23c49c614b0447ddd9c8c0dce3e7f22803991ecf7bbdab11326785ef6cba",
        "domain": "numina",
        "question": "Exercise 2. Anne drove her car for an integer (and non-zero) number of hours, and traveled an integer number of kilometers, at a speed of $55 \\mathrm{~km} / \\mathrm{h}$. At the beginning of the trip, the odometer read abc kilometers, where abc is a 3-digit number such that $a \\geqslant 1$ and $a+b+c \\leqslant 7$. At the end of the trip, the odometer read cba kilometers. Determine all possible values of the number abc.",
        "answer": "106",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78317,
        "difficulty": null
      },
      {
        "id": "3bddbb6da98cce4a675e82945774a96ff19ca8c5b6d28a89fa5e9d042c2c8756",
        "domain": "numina",
        "question": "A magpie cooked porridge and fed her chicks. The third chick received as much porridge as the first two combined. The fourth one received as much as the second and third. The fifth one received as much as the third and fourth. The sixth one received as much as the fourth and fifth. The seventh one did not get any - the porridge ran out! It is known that the fifth chick received 10 grams of porridge. How much porridge did the magpie cook?\n\n#",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70403,
        "difficulty": null
      },
      {
        "id": "74f1c80c213ac0292ebfebd355175eccb730d0bbc30ad1b728d2c5e4be89d27f",
        "domain": "numina",
        "question": "PROBLEM 4. Let the sequence $\\left(x_{n}\\right)_{n \\geq 1}$, defined by $x_{1}=3$ and $x_{n}=x_{n-1}+2 n+1$, for $n \\geq 2$.\n\na) Determine the general term of the sequence;\n\nb) Calculate $\\lim _{n \\rightarrow \\infty} n\\left(\\ln 2+\\ln \\left(\\sum_{k=1}^{n} \\frac{1}{x_{2 k-1}}\\right)\\right)$.[^0]\n\n## NATIONAL MATHEMATICS OLYMPIAD  Local stage - 14.02. 2015  GRADING SCALE - 11th Grade",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179474,
        "difficulty": null
      },
      {
        "id": "52bdbc417e1b3ba2cbc1b955dedef982788eb5ad9e285f70aad8f350c71575b9",
        "domain": "numina",
        "question": "23. In $\\triangle \\mathrm{ABC}, \\angle \\mathrm{CAB}=30^{\\circ}$ and $\\angle \\mathrm{ABC}=80^{\\circ}$. The point $\\mathrm{M}$ lies inside the triangle such that $\\angle \\mathrm{MAC}=10^{\\circ}$ and $\\angle \\mathrm{MCA}=30^{\\circ}$. Find $\\angle \\mathrm{BMC}$ in degrees.",
        "answer": "110",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160279,
        "difficulty": null
      },
      {
        "id": "f4124b2d6b76e55835ab6a5c96fa89224ebe0281d0dcb838682be9c6d04175ef",
        "domain": "numina",
        "question": "Example 3. Find $\\lim _{x \\rightarrow 4} \\frac{x^{2}-6 x+8}{x-4}$.",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32581,
        "difficulty": null
      },
      {
        "id": "88ded8f8d3a0bccd406f0af081afb5b7c9bb3d5b903e3924a0d80bf172f591bd",
        "domain": "numina",
        "question": "11. (20 points) Given the quadratic functions\n$$\ny_{1}=a x^{2}+4 x+b \\text { and } y_{2}=b x^{2}+4 x+a\n$$\n\nboth have minimum values, denoted as $m$ and $n$ for $y_{1}$ and $y_{2}$, respectively.\n(1) If $m+n=0$, prove that for any real number $x$, $y_{1}+y_{2} \\geqslant 0$;\n(2) If $m$ and $n$ are both greater than 0, and $m n=2$, let $M$ be the larger of $m$ and $n$. Find the minimum value of $M$.",
        "answer": "\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93771,
        "difficulty": null
      },
      {
        "id": "8fce58e57d12a573a02ff9354b653b75b218f8d1f0b0062bb4c0d779d9c477e0",
        "domain": "numina",
        "question": "4-7. Along a straight alley, 100 lamps are placed at equal intervals, numbered sequentially from 1 to 100. At the same time, from different ends of the alley, Petya and Vasya started walking towards each other at different constant speeds (Petya from the first lamp, Vasya from the hundredth). When Petya was at the 22nd lamp, Vasya was at the 88th lamp. At which lamp will their meeting occur? If the meeting occurs between two lamps, indicate the smaller number of these two.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7d113eccc6d3ed32fb40g-03.jpg?height=485&width=1865&top_left_y=1131&top_left_x=107)",
        "answer": "64",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14070,
        "difficulty": null
      },
      {
        "id": "8824fc9b1e235cd94bed8a4da34fcd9704f76d21f10228fda7ded2ce89777320",
        "domain": "numina",
        "question": "G4.4 If $x_{1}, x_{2}, x_{3}, x_{4}$ and $x_{5}$ are positive integers that satisfy $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=x_{1} x_{2} x_{3} x_{4} x_{5}$, that is the sum is the product, find the maximum value of $x_{5}$.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153315,
        "difficulty": null
      },
      {
        "id": "b123a85557a28aeecdea8c1254fae5b42dfa1f0992e73fe39cb8e0f4d6b10ab6",
        "domain": "numina",
        "question": "## Task 18/76\n\nGiven an annulus and $n$ circles within it such that each of them touches the inner and outer boundary circles and two other circles. How large is $n$ if the ratio of the annulus area $A_{1}$ to the sum of the $n$ circle areas $A_{2}$ is $\\sqrt{2}$?",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 182473,
        "difficulty": null
      },
      {
        "id": "238b5edcb43ecbb8edbca56055aa718ae7bc5ffd64bc9b73bb82503544071c3f",
        "domain": "numina",
        "question": "12.1.1. In a box, there are 6 identical pairs of black gloves and 4 identical pairs of beige gloves. Find the probability that two randomly drawn gloves form a pair.",
        "answer": "\\frac{47}{95}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 30422,
        "difficulty": null
      },
      {
        "id": "cdd9a5933bad509aef885ce24779ab1468da57ea6feab272be97f56dacccca8d",
        "domain": "numina",
        "question": "6. $P$ is the midpoint of the height $V O$ of the regular quadrilateral pyramid $V-A B C D$. If the distance from point $P$ to the side face is 3, and the distance from point $P$ to the base is 5, then the volume of the regular quadrilateral pyramid is",
        "answer": "750",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97677,
        "difficulty": null
      },
      {
        "id": "dd195ffdbc37685274f55b48f7c1b02796032a2d24ff727906285b1566ef6c1d",
        "domain": "numina",
        "question": "Task B-4.4. Determine all natural numbers $a$ for which the number $a^{3}+1$ is a power of 3.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88215,
        "difficulty": null
      },
      {
        "id": "0a5197603c66907a91c47f3931ffaf66f91a1093f21fa8519b934736df469ce7",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nSuppose that $O$ is the origin. Points $P(a, b)$ and $Q(c, 1)$ are in the first quadrant with $a=2 c$. If the slope of $O P$ is $t$ and the slope of $O Q$ is 1 , what is the slope of $P Q$ ?",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194211,
        "difficulty": null
      },
      {
        "id": "624a815134b3944dbbc51061f98d2095dd1d1429b4bf1fcfc545a4600541d33a",
        "domain": "numina",
        "question": "## 258. Math Puzzle $11 / 86$\n\nHans is supposed to nail a $90 \\mathrm{~cm}$ wide gap in a wooden fence. He has eight $10 \\mathrm{~cm}$ wide and sixteen $8 \\mathrm{~cm}$ wide boards of suitable length. A saw, with which he could split the wood lengthwise, is not available to him.\n\nHow many and which options does Hans have to close the opening without any gaps?",
        "answer": "1)5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181928,
        "difficulty": null
      },
      {
        "id": "7ec86c9a3b554e10b6c4d58729d9df8fa78573cf81d5bca5b85b5329d930f327",
        "domain": "numina",
        "question": "Example 6 (2008 National High School Mathematics Joint Competition, First Test) Let $f(x)$ be a function defined on $\\mathbf{R}$. If $f(0)=2008$, and for any $x \\in \\mathbf{R}$, it satisfies $f(x+2)-f(x) \\leqslant 3 \\cdot 2^{x}, f(x+6)-f(x) \\geqslant 63 \\cdot 2^{x}$, then $f(2008)=$ $\\qquad$ .",
        "answer": "2^{2008}+2007",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107986,
        "difficulty": null
      },
      {
        "id": "2172c7c956e8511b33e89dc4422bf5a9253cee9da9b6255a981c069a1b7f11e8",
        "domain": "numina",
        "question": "9.3. How many solutions in integers $x, y$ does the inequality $|y-x|+|3 x-2 y| \\leq a$ have a) for $a=2$; b) for $a=20$?",
        "answer": "13",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 21968,
        "difficulty": null
      },
      {
        "id": "8d471b40b75fcfd52c4acdbed6a2513426197561702ba0a591ab25851902213c",
        "domain": "numina",
        "question": "6. Several cells on a $14 \\times 14$ board are marked. It is known that no two of the marked cells are in the same row and the same column, and also that a knight can, starting from any marked cell, reach any other marked cell via marked cells. What is the maximum possible number of marked cells?",
        "answer": "14",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 14660,
        "difficulty": null
      },
      {
        "id": "a8338beb5f6a84a03817357b9cadb2e74ad6d15ca2de81474302a8344e853d37",
        "domain": "numina",
        "question": "## Zadatak B-1.6.\n\nRiješite jednadžbu\n\n$$\n\\frac{x^{4}-4 x^{3}+8 x^{2}-16 x+16}{x^{4}-16}=\\frac{1}{1 \\cdot 2}+\\frac{1}{2 \\cdot 3}+\\frac{1}{3 \\cdot 4}+\\cdots+\\frac{1}{25 \\cdot 26}\n$$",
        "answer": "102",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88126,
        "difficulty": null
      },
      {
        "id": "4951e0cc4f174a2439035195d3c99ff47fc4556b8d13da6eff2dde938e03a486",
        "domain": "numina",
        "question": "8. The value of the algebraic expression $\\frac{1}{1 \\cdot 3}+\\frac{1}{3 \\cdot 5}+\\cdots+\\frac{1}{(2 n-1)(2 n+1)}+\\cdots+\\frac{1}{255 \\cdot 257}$ is",
        "answer": "\\frac{128}{257}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104253,
        "difficulty": null
      },
      {
        "id": "c4c3a6cc0b626089c5dd28ae9781dc944f1e9ec3cda5323413a1ca658e53ec13",
        "domain": "numina",
        "question": "From point $A$, a passenger train travels towards $B$ at a speed of $30 \\mathrm{~km}$ per hour. A fast train, also departing from $A$ but later, travels at a speed of $60 \\mathrm{~km}$ per hour and catches up with the passenger train in $B$ under normal conditions. The passenger train, after covering ${ }^{2} / 3$ of its journey, is forced to continue at half its original speed due to a technical failure, which results in the fast train catching up $27^{1} / 9 \\mathrm{~km}$ before $B$. What is the distance between $A$ and $B$?",
        "answer": "122",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 173598,
        "difficulty": null
      },
      {
        "id": "33109e090a3de728c988767d7f657f482ddfb52ece8dde39bd1f168b32e36758",
        "domain": "numina",
        "question": "6. (3 points) Xiaofang participated in an English speaking competition, and the scores from ten judges are: 95 points, 94 points, 91 points, 88 points, 91 points, 90 points, 94 points, 93 points, 91 points, 92 points. If the highest score and the lowest score are removed, Xiaofang's final average score is $\\qquad$ points.",
        "answer": "92",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92455,
        "difficulty": null
      },
      {
        "id": "7badc8697ae4ceccc908595b406c1fd1e48d4749a12a8d8cc67e6330ecce14fb",
        "domain": "numina",
        "question": "9. For what values of the parameter a does the equation $f(x)=p(x)$ have one solution, if $f(x)=$ $\\left|\\frac{2 x^{3}-5 x^{2}-2 x+5}{(1.5 x-3)^{2}-(0.5 x-2)^{2}}\\right|, \\quad p(x)=|2 x+5|+a$. If there are more than one value of the parameter, then in the answer, specify their sum.\n\n#",
        "answer": "-10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 21811,
        "difficulty": null
      },
      {
        "id": "2180b3d9ea1baabebb3fef4698b5a5def46e308943690a4cae2ae654e79b44aa",
        "domain": "numina",
        "question": "## Task 4 - 340734\n\nA quadrilateral is called convex if and only if all its diagonals lie entirely within the area of the polygon.\n\nHow many diagonals does a convex 1995-gon have in total? Justify the number you provide!",
        "answer": "1987020",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183886,
        "difficulty": null
      },
      {
        "id": "a287d03b13d26da5a07a1e67abbf91cd77c6ef09509bee82f5ab34b8ea2aa598",
        "domain": "numina",
        "question": "Given is an unknown sequence $a_{1}, a_{2}, a_{3}, \\ldots$ of integers that satisfies the following property: for every prime number $p$ and every positive integer $k$ it holds that\n\n$$\na_{p k+1}=p a_{k}-3 a_{p}+13\n$$\n\nDetermine all possible values of $a_{2013}$.",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 195280,
        "difficulty": null
      },
      {
        "id": "9099894e43085f443bdb4dd5bd6ccef813788d46a89d980668edcc9fa01c6649",
        "domain": "numina",
        "question": "Task 1. We have two jugs, one holds exactly 3 liters, and the other holds exactly 5 liters. Can we measure exactly 4 liters using the two jugs?",
        "answer": "4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 188633,
        "difficulty": null
      },
      {
        "id": "2988047473b8832e919967868995a6c7e97749c0217e071e2c3dbd5fe00cf7f2",
        "domain": "numina",
        "question": "8. Draw a line with an inclination angle of $\\frac{\\pi}{4}$ through the point $P(11,0)$, intersecting the parabola $y^{2}=4 x$ at points $R$ and $Q$. Then draw a line parallel to $R Q$ intersecting the parabolic arc $\\overparen{R O Q}$ at points $M$ and $N$ (as shown in the figure). Find the maximum value of the area of $\\triangle P M N$.",
        "answer": "22",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 141482,
        "difficulty": null
      },
      {
        "id": "43c4f0caf39e6b020856e4f6e5d44d30977f0f00a05d0d8792272dced9dd0af8",
        "domain": "numina",
        "question": "Problem 7.6.1. In a row, there are 10 boxes. These boxes contain balls of two colors: red and blue. In some boxes, all the balls may be of the same color; there are no empty boxes. It is known that in each subsequent box (from left to right), there are no fewer balls than in the previous one. It is also known that there are no two boxes with the same set of red and blue balls. How many blue and how many red balls are in the rightmost box, if the total number of red balls in all the boxes is 11 and the total number of blue balls is 13?",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16023,
        "difficulty": null
      },
      {
        "id": "c427ab864316f236f0f0595024a652aaae409fd83e31079382fe302aa43e7e38",
        "domain": "numina",
        "question": "Ha az előbbi feladatban azt is tudni akarjuk, hogy a hamis pénz könnyebb-e, vagy nehezebb a többinél, hány pénzdarab a legtöbb, amennyi közül még három mérés elegendő ennek a kérdésnek az eldöntéséhez?\n\nIf in the previous task we also want to know whether the counterfeit coin is lighter or heavier than the others, how many coins is the maximum number from which three measurements are still sufficient to determine the answer to this question?",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 174939,
        "difficulty": null
      },
      {
        "id": "ad23f5fc7519046974c59394075a0221da7148d09a85177d65a33b3f83385fb1",
        "domain": "numina",
        "question": "## Task 1 - 040831\n\nIf the digits of a two-digit number $n$ are reversed, the resulting number is $\\frac{8}{3}$ times as large as $n$. The number $n$ is to be determined.",
        "answer": "27",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183964,
        "difficulty": null
      },
      {
        "id": "e39fddbd845b92b08ca9b6a4f4eb98e366d87448efbb97775f44e3c78e79f8cb",
        "domain": "numina",
        "question": "10. Let the function $f(x)=4 x^{3}+b x+1(b \\in \\mathbf{R})$, for any $x \\in[-1,1]$, it holds that $f(x) \\geqslant 0$. Find the range of the real number $b$.",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115691,
        "difficulty": null
      },
      {
        "id": "c47fffaa083a30ac9006428dd52cf4f355b8fa543a345a0e288f2213b792784d",
        "domain": "numina",
        "question": "8. If positive integers $a, b, c$ satisfy\n$$\n2019 \\geqslant 10 a \\geqslant 100 b \\geqslant 1000 c \\text {, }\n$$\n\nthen the number of arrays $(a, b, c)$ is $\\qquad$ .",
        "answer": "574",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 137076,
        "difficulty": null
      },
      {
        "id": "2645803f2a1fcaee7c6981d54b1cb62a6dd221e9026d25e318817f0d7dd6b92c",
        "domain": "numina",
        "question": "9. (10 points) The teacher in the kindergarten brought 55 apples, 114 cookies, 83 chocolates to the class, and after distributing them equally, there were 3 apples, 10 cookies, and 5 chocolates left. The class has at most $\\qquad$ children.",
        "answer": "26",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100559,
        "difficulty": null
      },
      {
        "id": "b0aa9a4aabdb32d4a2a33a91b2595142498b74e2537a1fb8e4ac77aae7c09afb",
        "domain": "numina",
        "question": "2. Two squirrels had the same number of pine cones and the same number of cedar cones. In total, each squirrel had fewer than 25 cones. The first squirrel gathered as many pine cones as it already had and 26 cedar cones. It ended up with more pine cones than cedar cones. The second squirrel gathered as many cedar cones as it already had and ate 4 pine cones. It ended up with more cedar cones than pine cones. How many pine and cedar cones did each squirrel have initially?",
        "answer": "17",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4839,
        "difficulty": null
      },
      {
        "id": "a64a1d8459f141fb6130f45fca4f9c6a790bf26bd3f567083cdf016218b2e92c",
        "domain": "numina",
        "question": "4. Calculate: $2017 \\times 2016 + 2016 \\times 2014 - 2015 \\times 2016 - 2015 \\times 2017$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100774,
        "difficulty": null
      },
      {
        "id": "ad82c83571493eaead1255dcbb7a9cd4ca5150b234175e7ee8554bbb315b9bfe",
        "domain": "numina",
        "question": "## Task A-2.4.\n\nAll vertices of the hexagon $A B C D E F$ lie on a circle with diameter $\\overline{A D}$. The line $B F$ intersects the lines $A D$ and $C E$ at points $G$ and $H$ respectively. If $\\varangle F E H=56^{\\circ}, \\varangle D G B=124^{\\circ}$ and $\\varangle D E C=34^{\\circ}$, determine $\\varangle C E B$.",
        "answer": "22",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87978,
        "difficulty": null
      },
      {
        "id": "633a2cd7601944a0cef9d1a8bc969dc8f6332166fb5bedd0f6c5ae78bcb7869c",
        "domain": "numina",
        "question": "56. In the class, 12 subjects are studied. In how many ways can 5 different subjects be scheduled for Monday?",
        "answer": "9504",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44877,
        "difficulty": null
      },
      {
        "id": "0dbf8fe6e01e5d44763babaf62281d4c8e865d34b482db867a29d79418b67850",
        "domain": "numina",
        "question": "## [ Rectangles and Squares. Properties and Characteristics  Method of Coordinates\n\nOn the plane, a square $A B C D$ is given. Find the minimum of the ratio $\\frac{O A+O C}{O B+O D}$, where $O-$ is an arbitrary point on the plane.",
        "answer": "\\frac{1}{\\sqrt{2}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 53740,
        "difficulty": null
      },
      {
        "id": "db15a19e91c723fce13bf3f2c1c2fa3b86877b8861e031d367681bb2ddc2dfb2",
        "domain": "numina",
        "question": "14. (15 points) As shown in the figure, in a square $P Q R S$ with a side length greater than $20 \\mathrm{~cm}$, there is a largest circle $O$. If a point $T$ on the circumference of the circle is $8 \\mathrm{~cm}$ away from $P S$ and $9 \\mathrm{~cm}$ away from $P Q$, what is the radius of circle $O$ in centimeters?",
        "answer": "29",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 95802,
        "difficulty": null
      },
      {
        "id": "8c55247e535100816e11d00cf33f6595885e8893946a4746126a1615e7fff5f0",
        "domain": "numina",
        "question": "9. (16 points) Find the integer part of $\\sqrt{100+\\sqrt{101+\\cdots+\\sqrt{999}}}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131501,
        "difficulty": null
      },
      {
        "id": "59add37cc43cbcf8d299282e5dd7ac4fadc1e59b1d8b8b8eae640205fbf50d06",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a_{\\text {and }} b$.\n\n\\[\n\\begin{aligned}\n& a=4 p-q \\\\\n& b=p+2 q \\\\\n& |p|=5 \\\\\n& |q|=4 \\\\\n& (\\widehat{p, q})=\\frac{\\pi}{4}\n\\end{aligned}\n\\]",
        "answer": "90\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47412,
        "difficulty": null
      },
      {
        "id": "b0b94fc96ff11e3d344e5c5da4db3843091f6411bba29f592fe542d4786e735d",
        "domain": "numina",
        "question": "Point $A$ lies in the plane $\\alpha$, the orthogonal projection of segment $A B$ onto this plane is $1, A B=2$. Find the distance from point $B$ to the plane $\\alpha$.",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58991,
        "difficulty": null
      },
      {
        "id": "eac85abddec44801baa772fa5ed02b09e74a19fcb6152e6f2531469376465bb9",
        "domain": "numina",
        "question": "11.32. The area of a trapezoid is 1. What is the smallest value that the largest diagonal of this trapezoid can have?",
        "answer": "\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 27525,
        "difficulty": null
      },
      {
        "id": "dd700465cdc95c97ee170251f3cb23bdccc11f66b087a388efc62e0fed8e237e",
        "domain": "numina",
        "question": "276. Solve the equation:\n\n$$\n\\left(x^{2}-16\\right)(x-3)^{2}+9 x^{2}=0\n$$",
        "answer": "-1\\\\sqrt{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 34632,
        "difficulty": null
      },
      {
        "id": "8f930b5668d604db3de827f15788a6783941c6411082dd9ce404da357a590467",
        "domain": "numina",
        "question": "# Task 8.5\n\nOn the Island of Liars and Knights, a circular arrangement is called correct if each person standing in the circle can say that among their two neighbors, there is a representative of their tribe. Once, 2019 natives formed a correct arrangement in a circle. A liar approached them and said: \"Now we can also form a correct arrangement in a circle together.\" How many knights could there have been in the initial arrangement?\n\n## Number of points 7",
        "answer": "1346",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13484,
        "difficulty": null
      },
      {
        "id": "67ecf2699769a7ee3cf8ffcfb52a6daf8ec3fc2922cb6f5a7437fd0e2ce463e0",
        "domain": "numina",
        "question": "4. On an $8 \\times 8$ board, there is a beetle on each square. At a certain moment, the distribution of the beetles on the board changes: each beetle crawls either one square to the left or one square diagonally to the right and down. If a beetle cannot make either move without falling off the board, it stays on its square.\n\nHow many squares can become unoccupied at most due to this change?",
        "answer": "29",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 85867,
        "difficulty": null
      },
      {
        "id": "b96f878a21aee1ddeb6b76c8412a7756e8e60072639e68b73ac01b9245759470",
        "domain": "numina",
        "question": "10.8. A numismatist has 100 coins that look identical. He knows that among them, 30 are genuine and 70 are counterfeit. Moreover, he knows that the weights of all genuine coins are the same, while the weights of all counterfeit coins are different, and any counterfeit coin is heavier than a genuine one; however, the exact weights of the coins are unknown. There are balance scales without weights, on which it is possible to compare the weights of two groups consisting of the same number of coins in one weighing. What is the minimum number of weighings on these scales that the numismatist can guarantee to find at least one genuine coin?\n\n(C. Berlov, I. Bogdanov)",
        "answer": "70",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17141,
        "difficulty": null
      },
      {
        "id": "eccaee603cc7427bb175caaa0971dc526173adb9de9421f05cb1b88415d1a2bb",
        "domain": "numina",
        "question": "13.\n\nA box of hawthorns has more than one hundred grains. When counted in groups of 3, there is 1 extra; when counted in groups of 4, there are 2 extra; when counted in groups of 5, there are 3 extra; and when counted in groups of 6, there are 4 extra. The box of hawthorns has at most ( ) grains.",
        "answer": "178",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144739,
        "difficulty": null
      },
      {
        "id": "2e88ceffb332562521360f086af37276b256bc2b78b04e53561a65e15c7d8b13",
        "domain": "numina",
        "question": "9. (16 points) Let $x \\in [0, 2\\pi]$. Find the maximum value of the function\n$$\n\\begin{aligned}\nf(x)= & \\sqrt{4 \\cos ^{2} x + 4 \\sqrt{6} \\cos x + 6} + \\\\\n& \\sqrt{4 \\cos ^{2} x - 8 \\sqrt{6} \\cos x + 4 \\sqrt{2} \\sin x + 22}\n\\end{aligned}\n$$",
        "answer": "2(\\sqrt{6}+\\sqrt{2})",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141510,
        "difficulty": null
      },
      {
        "id": "ce33240ef5e2ebcda28e181411ba4a13dc313f89bd9dbd53cc8e79375b3a7f25",
        "domain": "numina",
        "question": "Task 3. (15 points) An educational center \"Young Geologist\" received an object for research consisting of about 300 monoliths (a container designed for 300 monoliths, which was almost completely filled). Each monolith has a specific name (sandy loam or clayey loam) and genesis (marine or lake-glacial deposits). The relative frequency (statistical probability) that a randomly selected monolith will be sandy loam is $\\frac{1}{8}$. The relative frequency that a randomly selected monolith will be marine clayey loam is $\\frac{22}{37}$. How many monoliths of lake-glacial genesis does the object contain, if there are no marine sandy loams among the sandy loams?",
        "answer": "120",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7303,
        "difficulty": null
      },
      {
        "id": "2462620041884eb875ec6501b6b0614a12c517b56b14965161a3f7f0c18771f2",
        "domain": "numina",
        "question": "9. A gardener is preparing to plant a row of 20 trees, with two types of trees available: maple trees or sycamore trees. The number of trees between any two maple trees (not including these two maple trees) cannot be equal to 3. How many maple trees can there be at most among the 20 trees? $\\qquad$",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 94631,
        "difficulty": null
      },
      {
        "id": "98229bf393b63fe9ad8f2edb691fec1c5e884863dc6fe351f0b63131de7bcf6b",
        "domain": "numina",
        "question": "2. Graphically solve the equation $\\log _{2}(x-1)+1=-\\frac{3}{2} x+4$. Verify the solution by calculation.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163745,
        "difficulty": null
      },
      {
        "id": "24b7f54454677ac15b63dd0ea023d5c4d80cdb48001122a947ecd06ba435cb54",
        "domain": "numina",
        "question": "10.3. The sum $1+\\frac{1}{2}+\\frac{1}{3}+\\ldots+\\frac{1}{45}$ is represented as a fraction with the denominator $45!=1 \\cdot 2 \\cdots 45$. How many zeros (in decimal notation) does the numerator of this fraction end with?",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 22591,
        "difficulty": null
      },
      {
        "id": "d0b89e4af271c413235c138051e341a825fe84da40d4953f3929584b5740d960",
        "domain": "numina",
        "question": "10. Consider the expression $1 * 2 * 3 * 4 * 5 * 6$.\nEach star in the expression is to be replaced with either ' + ' or ' $x$ '.\n$N$ is the largest possible value of the expression. What is the largest prime factor of $N$ ?",
        "answer": "103",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156438,
        "difficulty": null
      },
      {
        "id": "e495555f53acdddcd528f4a035640a544b520eda0110800971a5a7807ff70d83",
        "domain": "numina",
        "question": "Example 8 Color each vertex of a quadrilateral pyramid with one color, and make the endpoints of the same edge have different colors. If only 5 colors are available, how many different coloring methods are there?",
        "answer": "420",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104263,
        "difficulty": null
      },
      {
        "id": "17a830017722596950154160734a26d03bf47d0cd9ec7c89a5ad08d8c8a4b30a",
        "domain": "numina",
        "question": "14. A ball was added to an urn containing one white ball - either white or black (with equal probabilities of selection). After this, one ball was randomly drawn from the urn. It turned out to be white. What is the conditional probability that the remaining ball in the urn is also white?",
        "answer": "\\frac{2}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 33498,
        "difficulty": null
      },
      {
        "id": "561b89cee9df5266b2cae789a024261ffe5ee18be95070353d70657afae62655",
        "domain": "numina",
        "question": "65. Find two numbers whose sum is 20 and product is 96.",
        "answer": "128",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 37716,
        "difficulty": null
      },
      {
        "id": "d4c618cfa2bf251d2f90167c86fdaa7f40354026792af0fe224d21d75de4e359",
        "domain": "numina",
        "question": "29.1. Find the three-digit number $\\overline{a b c}$, if it is known that\n\n$$\n\\begin{gathered}\n8 a+5 b+c=100 \\\\\na+b+c=20\n\\end{gathered}\n$$",
        "answer": "866",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 44947,
        "difficulty": null
      },
      {
        "id": "a41e6e856ecf6f937b8c8a903acb267241e88cec30a07c06551ab1b8183a036e",
        "domain": "numina",
        "question": "2B. In a certain populated place, all telephone numbers consist of six digits arranged in strictly ascending or strictly descending order, with the first digit in the number not being 0. What is the maximum number of telephone numbers that can exist in this place?",
        "answer": "294",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 190510,
        "difficulty": null
      },
      {
        "id": "0f63f083fe544829ca311c21146b0b7a89fa4fd025ac3b98e15fdf46e99b34d3",
        "domain": "numina",
        "question": "12 different items are distributed among 3 people so that each person gets 4 items. In how many ways is this possible?",
        "answer": "34650",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 166442,
        "difficulty": null
      },
      {
        "id": "003e2db356e71c6b17dcdf46aa575ec1a755eb2eb6a82b6e5dd02b2f4dba78fd",
        "domain": "numina",
        "question": "29. When from the numbers from 1 to 333 Tanya excluded all numbers divisible by 3 but not divisible by 7, and all numbers divisible by 7 but not divisible by 3, she ended up with 215 numbers. Did she solve the problem correctly?",
        "answer": "205",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39427,
        "difficulty": null
      },
      {
        "id": "20ecfc754dc6d321ff12469606dd80df578f0be67b9266ba75ac704f986e27cc",
        "domain": "numina",
        "question": "Three, (25 points) Find all positive integers such that it is 224 times the sum of its digits.\n\n\n\nThe above text has been translated into English, retaining the original text's line breaks and format.",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 119471,
        "difficulty": null
      },
      {
        "id": "5f7d8c8492ffe2225cb342f73513bfdc213eb9bdafdf0064429627a63538e3aa",
        "domain": "numina",
        "question": "3. Point $O$ is the center of the circumcircle of triangle $ABC$. On the circumcircle of triangle $BOC$, outside triangle $ABC$, point $X$ is chosen. On the rays $XB$ and $XC$ beyond points $B$ and $C$, points $Y$ and $Z$ are chosen respectively such that $XY = XZ$. The circumcircle of triangle $ABY$ intersects side $AC$ at point $T$. Find the angle $\\angle Y T Z$.",
        "answer": "180",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8233,
        "difficulty": null
      },
      {
        "id": "95fe67b2ca7ede6c8a580f0312e73a75fe9d100dbda9b9a58f14d271dfd72c8d",
        "domain": "numina",
        "question": "25. One day, the owner put more than 60 identical chess pieces into some identical boxes, with only one box being empty. The mischievous Tom cat took one piece from each of the boxes that had pieces and put them into the empty box. The clever Jerry mouse rearranged the boxes, and as a result, the position of each box and the number of pieces in each box looked the same as before. Therefore, there are $\\qquad$ boxes.",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 112589,
        "difficulty": null
      },
      {
        "id": "27a7d60b6fc9bc6edecc75fc652c23f263efbf5782c019c7bb5b68597c56d52d",
        "domain": "numina",
        "question": "2. The minimum value of the function $f(x)=\\frac{(\\sin x+1)(\\cos x+1)}{\\sin x \\cos x}$ on $\\left(0, \\frac{\\pi}{2}\\right)$ is",
        "answer": "3+2\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101229,
        "difficulty": null
      },
      {
        "id": "9b1cb57d7a23befd10c8ea4e3d8d1202f2560f9446f32cd2093fd8221ed73240",
        "domain": "numina",
        "question": "\\section*{Problem 2 - 121232}\n\nIn space, four points \\(P_{1}, P_{2}, P_{3}\\) and \\(P_{4}\\) are given, which do not lie in the same plane. Determine the number of all those planes that are equidistant from these four points.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 185830,
        "difficulty": null
      },
      {
        "id": "4582b9074886aa97ed96878206e07d3435046fcb0e42766897bb0bf7f2cabe4b",
        "domain": "numina",
        "question": "[ Tiling with domino bones and tiles ] ..... ]\n[ Examples and counterexamples. Constructions ]\n\nOn a grid sheet of size 100×100, several pairwise non-overlapping cardboard isosceles right triangles with a leg of 1 were placed; each triangle occupies exactly half of one of the cells. It turned out that each unit segment of the grid (including boundary segments) is covered by exactly one leg of a triangle. Find the maximum possible number of cells that do not contain any triangle.",
        "answer": "2450",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 63857,
        "difficulty": null
      },
      {
        "id": "044f812ee059df4e902932e0ac2e5e930d35765fe9c10a32a12fe97d218c422a",
        "domain": "numina",
        "question": "7. Given that each carbon pen costs 1 yuan 8 jiao, each notebook costs 3 yuan 5 jiao, and each stationery box costs 4 yuan 2 jiao, Jingjing bought these three stationery items and spent exactly 20 yuan. How many notebooks did she buy? $\\qquad$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120999,
        "difficulty": null
      },
      {
        "id": "b532779b35809fd1590990186fcf68f05580e5b3c7c25d409c7345ab48d6271f",
        "domain": "numina",
        "question": "6. Find the largest constant $K \\in \\mathbb{R}$ with the following property: if $a_{1}, a_{2}, a_{3}, a_{4}>0$ are such that for all $i, j, k \\in \\mathbb{N}, 1 \\leqslant i<j<k \\leqslant 4$, we have $a_{i}^{2}+a_{j}^{2}+a_{k}^{2} \\geqslant 2\\left(a_{i} a_{j}+a_{j} a_{k}+a_{k} a_{i}\\right)$, then\n\n$$\na_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2} \\geqslant K\\left(a_{1} a_{2}+a_{1} a_{3}+a_{1} a_{4}+a_{2} a_{3}+a_{2} a_{4}+a_{3} a_{4}\\right)\n$$\n\n(Dušan Đukić)",
        "answer": "\\frac{11}{7}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 161231,
        "difficulty": null
      },
      {
        "id": "4461d590286bdddb04308eba226a9ac4ae84f72f13158a6b0f69265a3f021091",
        "domain": "numina",
        "question": "Calculate $\\frac{1}{3}$ in $\\mathbb{Z}_{5}$.\n\n untranslated text:\nCalculer $\\frac{1}{3}$ dans $\\mathbb{Z}_{5}$.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78909,
        "difficulty": null
      },
      {
        "id": "b1873f4935f14f7629057a791cd9c18840f2ec1afc5a53d54fc0de415a647c11",
        "domain": "numina",
        "question": "6.8 In the school football championship, each team must play a match against every other team. Each match's winning team gets 2 points, a draw gives each team 1 point, and the losing team gets 0 points. It is known that one team scored the most points (every other team scored fewer points than this team), but it won fewer matches than any other team. How many teams participated at the minimum?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 122147,
        "difficulty": null
      },
      {
        "id": "d2c8fc117cc32d999a777ba8da720b8719629fc9fff5718addf9eb7aefc229a5",
        "domain": "numina",
        "question": "A sphere of radius 4 with center at point $Q$ touches three parallel lines at points $F, G$, and $H$. It is known that the area of triangle $Q G H$ is $4 \\sqrt{2}$, and the area of triangle $F G H$ is greater than 16. Find the angle $G F H$.",
        "answer": "67.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62419,
        "difficulty": null
      },
      {
        "id": "c93153b77430bc6592b18fe3269d704ac60e00da5c89dba3b0b93ae8b8061507",
        "domain": "numina",
        "question": "11. Given the quadratic function $y=a x^{2}+b x+c$ whose graph intersects the $x$-axis at points $A$ and $B$, and whose vertex is $C$.\n(1) If $\\triangle A B C$ is a right triangle, find the value of $b^{2}-4 a c$.\n(2) Consider the quadratic function\n$$\ny=x^{2}-(2 m+2) x+m^{2}+5 m+3\n$$\n\nwhose graph intersects the $x$-axis at points $E$ and $F$, and intersects the graph of the linear function $y=3 x-1$ at two points, with the point having the smaller $y$-coordinate denoted as point $G$.\n(i) Express the coordinates of point $G$ in terms of $m$;\n(ii) If $\\triangle E F G$ is a right triangle, find the value of $m$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143307,
        "difficulty": null
      },
      {
        "id": "90ee00f0f1270a81d444f7347bed773c0eb6e365c8427e399b0e91abc6028b9a",
        "domain": "numina",
        "question": "[ Tangent lines and tangent circles ] Described quadrilaterals $\\quad]$\n\nTwo circles of radii 4 and 3 touch each other externally. Common external tangents $P Q$ and $R S$ are drawn to these circles such that points $P$ and $S$ lie on the circle of the larger radius, and points $Q$ and $R$ lie on the circle of the smaller radius. Find the radius of the circle that touches segments $R S$, $S P$, and $P Q$.",
        "answer": "\\frac{24}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65008,
        "difficulty": null
      },
      {
        "id": "5de7bbe7ded6366d45d315073737c71f3201d1e6d87ea94bab9dfdbdb63d1de3",
        "domain": "numina",
        "question": "Task B-3.5. (20 points) Two equal chessboards ($8 \\times 8$ squares) have a common center and one completely covers the other. One of them is rotated around the center by $45^{\\circ}$. Determine the area of the intersection of all black squares of one board with all black squares of the other board if the area of one square is 1.",
        "answer": "32(\\sqrt{2}-1)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88114,
        "difficulty": null
      },
      {
        "id": "e6c448c773d8800a03aca99b950ae0b64e6a1c98adbe72c2722496f5ea9dcd04",
        "domain": "numina",
        "question": "24.3.5 $\\star$ * 64 points arranged in an 8 by 8 grid, how many lines are needed at minimum to separate them?",
        "answer": "14",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 127224,
        "difficulty": null
      },
      {
        "id": "37b24d2636fc011eab92e1ec8b7d938ac7824d9d72a9634425e17de4778e7c37",
        "domain": "numina",
        "question": "## Problem Statement\n\nBased on the definition of the derivative, find $f^{\\prime}(0)$:\n\n$$\nf(x)=\\left\\{\\begin{array}{c}\ne^{\\sin \\left(x^{\\frac{3}{2}} \\sin \\frac{2}{x}\\right)}-1+x^{2}, x \\neq 0 \\\\\n0, x=0\n\\end{array}\\right.\n$$",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47777,
        "difficulty": null
      },
      {
        "id": "cd2e368160833890261413fab2a916918a1f8b3eceb97e3195de05c52f2a1344",
        "domain": "numina",
        "question": "19. (3 points) The Hope Cup Committee in a certain area is arranging examination rooms for candidates participating in the Hope Cup examination. If each examination room is arranged for 30 candidates, there will be one examination room with 26 candidates; if each examination room is arranged for 26 candidates, there will be one examination room with 20 candidates, and it will require 9 more examination rooms than the previous arrangement. The number of candidates participating in the examination in this area is $\\qquad$.",
        "answer": "1736",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102083,
        "difficulty": null
      },
      {
        "id": "29a6e885f0202214a5997e6e3ce65e6a57c2895227ec0ea0bc3ea426e399f58b",
        "domain": "numina",
        "question": "## Task 2\n\nMultiply the numbers 7 and 6. Subtract 5 from the product.\n\nWrite the equation and solve.",
        "answer": "37",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186475,
        "difficulty": null
      },
      {
        "id": "c05f96afebd1b18f64bdeebea24c05a4a35b4bd93b625de269fb6a54633b977e",
        "domain": "numina",
        "question": "10. Evaluate\n$$\n\\int_{-\\infty}^{\\infty} \\frac{1-x^{2}}{1+x^{4}} d x\n$$",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 152180,
        "difficulty": null
      },
      {
        "id": "e51df3a8ebb0b96967d33ad7bccaff0d871bf492201bf9d9a2a0e5150a08f30c",
        "domain": "numina",
        "question": "3. Given that $a, b$ are real numbers, if the quadratic function $f(x)=x^{2}+a x+b$ satisfies $f(f(0))=f(f(1))=0$, and $f(0) \\neq f(1)$, then the value of $f(2)$ is $\\qquad$ .",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120148,
        "difficulty": null
      },
      {
        "id": "fd8406797810390a3b39bb2f08c4a8c633a9fd18c7dfebcbf0b6da3c4cbbe231",
        "domain": "numina",
        "question": "7. Given a tetrahedron $ABCD$ where one edge length is 3, and the other five edge lengths are all 2. Then the radius of its circumscribed sphere is $\\qquad$",
        "answer": "\\frac{\\sqrt{21}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 125392,
        "difficulty": null
      },
      {
        "id": "2ebf2e44dd653d951abfd67de5d3ef0096afca292292243c92cdb691aba61ac2",
        "domain": "numina",
        "question": "Exercise 8. Let $x, y, z$ be three real numbers satisfying $x+y+z=2$ and $xy+yz+zx=1$. Determine the maximum value that $x-y$ can take.",
        "answer": "\\frac{2}{\\sqrt{3}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 77680,
        "difficulty": null
      },
      {
        "id": "2e0fc88a8c4bb1cc7c8d10bdd68dd459b155e5eb6575623854923f2fcbd9efc3",
        "domain": "numina",
        "question": "Example 8 Find the maximum value of the function\n$$\nf(x)=\\sqrt{-x^{2}+10 x-9}+\\sqrt{-x^{2}+68 x-256}\n$$",
        "answer": "3\\sqrt{35}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105475,
        "difficulty": null
      },
      {
        "id": "19cb5c5df8e778b4fae475e0902e0ddb8aa3a155f53ea3ba291066a1200bdc45",
        "domain": "numina",
        "question": "8. The function $f(x)$ defined on the set $\\left\\{x \\in \\mathbf{Z}_{+} \\mid 1 \\leqslant x \\leqslant 12\\right\\}$ satisfies\n$$\n|f(x+1)-f(x)|=1(x=1,2, \\cdots, 11),\n$$\n\nand $f(1) 、 f(6) 、 f(12)$ form a geometric sequence. If $f(1)$ $=1$, then the number of different functions $f(x)$ that satisfy the conditions is $\\qquad$",
        "answer": "155",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 135830,
        "difficulty": null
      },
      {
        "id": "7dcc674b87ef80962d46a6ace4779f656c4fcc37e4280f835d3f5aa93bb5d8f5",
        "domain": "numina",
        "question": "6. (48th Slovenian Mathematical Olympiad) Find all prime numbers $p$ such that $p+28$ and $p+56$ are also prime.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 141776,
        "difficulty": null
      },
      {
        "id": "432ed92d9412eb45e37d442f0c6625483a309dbd10d6e1882a6e6d12a898b7fe",
        "domain": "numina",
        "question": "10. As shown in the figure, $\\triangle A B C$ is an isosceles right triangle, $E$ is a point on side $B C$ such that $B E=3 C E$, and points $D, A, F$ are collinear. Let the height from $D$ to $B E$ in $\\triangle D B E$ be $h_{1}$, and the height from $F$ to $E C$ in $\\triangle F E C$ be $h_{2}$. We have $3 h_{1}+h_{2}=3$ cm. The sum of the areas of $\\triangle D B E$ and $\\triangle F E C$ is 6 square cm, then the area of $\\triangle A B C$ is $\\qquad$ square cm.",
        "answer": "64",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114551,
        "difficulty": null
      },
      {
        "id": "bdf2a3eff659e1300ef867e373428e26dde3d1fbcb0a6c772b885edb8a74c270",
        "domain": "numina",
        "question": "Problem 5. $DABC$ is a triangular pyramid (Fig. 5). $\\overline{AK}=\\overline{KD}, \\overline{BP}=\\overline{PC}, \\overline{DM}=0.4 \\overline{DC}$. Find the area $S$ of the section of the pyramid by the plane $KMP$, if the vertex $A$ is at a distance $h=1$ from the plane $KMP$ and the volume of the pyramid $DABC$ is 5.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 37642,
        "difficulty": null
      },
      {
        "id": "c3655816e1175b7ca435bd9f71a1382b47e38f966c4ff78938af263abbb62463",
        "domain": "numina",
        "question": "3.11 Apples of the first grade for a total of 228 rubles and apples of the second grade for a total of 180 rubles were delivered to the store. During unloading, the delivered apples were accidentally mixed. It was found that if all the apples are now sold at one price - 90 kopecks lower than the price per kilogram of first-grade apples, the planned amount will be received. How many kilograms of apples were delivered, if it is known that there were 5 kg more of second-grade apples than first-grade apples?\n\n## Ratios between natural numbers",
        "answer": "85",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49551,
        "difficulty": null
      },
      {
        "id": "59fd47000a25290a7da089c1185c985840f20af40604d628e9576a89b77228f6",
        "domain": "numina",
        "question": "13. Find the remainder when $(1!\\times 1)+(2!\\times 2)+(3!\\times 3)+\\cdots+(286!\\times 286)$ is divided by 2009 .",
        "answer": "2008",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160486,
        "difficulty": null
      },
      {
        "id": "b063d9f98ee619767530916a6a672c9d37390d47b2eaa12922a3916b3c5f71a8",
        "domain": "numina",
        "question": "15.5. (GDR, 74). Determine into how many parts a regular tetrahedron is divided by six planes, each of which passes through one edge and the midpoint of the opposite edge of the tetrahedron. Find the volume of each of the parts if the volume of the tetrahedron is 1.",
        "answer": "\\frac{1}{24}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29065,
        "difficulty": null
      },
      {
        "id": "abd978244291f151eaada09e4823e704c4cc1361ffd281bf62c87e67054a1aae",
        "domain": "numina",
        "question": "$\\underline{\\text { Folklore }}$\n\nFind the smallest natural $n$, for which the number $A=n^{3}+12 n^{2}+15 n+180$ is divisible by 23.",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69743,
        "difficulty": null
      },
      {
        "id": "3e1793cb51ab4bd33dad65a26935420a4a2d8d51464157a48abdf8d09a9fa7f2",
        "domain": "numina",
        "question": "8.1. Lesha wrote down a number, and then replaced the same digits with the same letters, and different digits with different letters. He ended up with the word NOVOSIBIRSK. Could the original number be divisible by 9?",
        "answer": "10203454638",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9743,
        "difficulty": null
      },
      {
        "id": "f7ef5b00abd86bbf511f70214d8aaf4fe81c1cf2031d2f5c59e2a06432fc01aa",
        "domain": "numina",
        "question": "Create a section on a sphere whose area is equal to one-third of the great circle's area, and on the intersecting surface, place a cone whose apex falls into the center of the sphere; if the volume of the cone is $480 \\mathrm{~m}^{3}$, what will be the radius of the sphere?",
        "answer": "11.90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 172383,
        "difficulty": null
      },
      {
        "id": "d1328165286ee3e516ebcaec9e61e5db4e678f7354d1b600db037b24303ff519",
        "domain": "numina",
        "question": "2. Given $a>0, x, y$ satisfy the constraint conditions $\\left\\{\\begin{array}{l}x \\geqslant 1, \\\\ x+y \\leqslant 3, \\\\ y \\geqslant a(x-3) .\\end{array}\\right.$ If the minimum value of $z=2 x+y$ is 1, then $a=$",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130182,
        "difficulty": null
      },
      {
        "id": "b12ac3848d7d4a0a57456655c8fd9cb12db9cd58d9f6b4834e283e363dc22568",
        "domain": "numina",
        "question": "8. (3 points) A large cube, four medium cubes, and four small cubes are assembled into a three-dimensional figure as shown in the diagram. It is known that the edge lengths of the large, medium, and small cubes are 5 cm, 2 cm, and 1 cm, respectively. What is the surface area of this three-dimensional figure in square centimeters?",
        "answer": "230",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102919,
        "difficulty": null
      },
      {
        "id": "dd18171ec70ee794f40a2f78ebdc609beda9e7738641179293d07bfde72b4f37",
        "domain": "numina",
        "question": "10th Mexico 1996 Problem A2 64 tokens are numbered 1, 2, ... , 64. The tokens are arranged in a circle around 1996 lamps which are all turned off. Each minute the tokens are all moved. Token number n is moved n places clockwise. More than one token is allowed to occupy the same place. After each move we count the number of tokens which occupy the same place as token 1 and turn on that number of lamps. Where is token 1 when the last lamp is turned on?",
        "answer": "32",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 149341,
        "difficulty": null
      },
      {
        "id": "4550ff910c57d6ade260a2e57e3d06ce1429042a4e6f5909507e833a5cad096c",
        "domain": "numina",
        "question": "4. To build a batch of identical houses with a total area of $2500 \\mathrm{~m}^{2}$, the cost of a $a \\mathrm{~m}^{2}$ house is the sum of material cost $100 p_{1} a^{\\frac{3}{2}}$ yuan, construction cost $100 p_{2} a$ yuan, and other various expenses $100 p_{3} a^{\\frac{1}{2}}$ yuan, where the numbers $p_{1} 、 p_{2} 、 p_{3}$ are three consecutive terms of a geometric sequence, their sum is 21, and their product is 64. If 63 such houses are built, the material cost will be less than the sum of the construction cost and other various expenses. To minimize the total cost, what is the maximum number of houses that can be built?",
        "answer": "156",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138878,
        "difficulty": null
      },
      {
        "id": "934032faef4f30157df6b9b20dc543daaee1330eaf7b6da3411b68a631e894a8",
        "domain": "numina",
        "question": "3. (3 points) In a certain country, there are 50 cities. The Ministry of Aviation requires that every two cities be connected by a two-way flight operated by exactly one airline, and that it should be possible to travel from any city to any other using the flights of each airline (possibly with layovers). What is the maximum number of airlines for which this is possible?\n\n## Answer: 25",
        "answer": "25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20998,
        "difficulty": null
      },
      {
        "id": "c412a39a8d7cb5d3d6f87eea2259cff883e2f209d906e501d59ab9f1abfa1f22",
        "domain": "numina",
        "question": "In how many ways can the nine digits from 1 to 9 be placed in a $3 \\times 3$ grid so that the sums of the rows and the sums of the columns are all equal?",
        "answer": "72",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 78921,
        "difficulty": null
      },
      {
        "id": "5b7b097ad342c818a2955dcddfaf9799f55016d308a7a3c9370e89da10bcf1c3",
        "domain": "numina",
        "question": "4. In the rebus\n\n$$\n\\mathbf{K}\\mathbf{O}>\\mathbf{H}>\\mathbf{A}>\\mathbf{B}>\\mathbf{U}>\\mathbf{P}>\\mathbf{y}>\\mathbf{C}\n$$\n\ndifferent letters represent different digits. How many solutions does the rebus have?",
        "answer": "0",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14483,
        "difficulty": null
      },
      {
        "id": "854d462c7f6e5c733061699eb73d6f0c9f0a434fb29f115b9b69a5ba26cbe34a",
        "domain": "numina",
        "question": "113. How many two-digit natural numbers are there in which the first digit is greater than the second?",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 36205,
        "difficulty": null
      },
      {
        "id": "128c94aeb2d0095e516fa006f3b76b13783b06e6ee77047dba7e54ed91b883e6",
        "domain": "numina",
        "question": "3. Given a trapezoid $A B C D$ with bases $A B$ and $C D$. Let $E$ be the midpoint of side $A B$, $F$ the midpoint of segment $D E$, and $G$ the intersection of segments $B D$ and $C E$. Express the area of trapezoid $A B C D$ in terms of its height $v$ and the length of segment $F G$, assuming that points $A, F, C$ lie on a straight line.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_1d8fc6c4c0f260a9dad0g-3.jpg?height=517&width=648&top_left_y=1723&top_left_x=710)",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75756,
        "difficulty": null
      },
      {
        "id": "d8af841fcde1fa07c1b009bfd2daa37f7e09f02e24f3b9a997a7e318f5fc0d53",
        "domain": "numina",
        "question": "8. Let the integer sequence $a_{1}, a_{2}, \\cdots, a_{10}$ satisfy $a_{10}=3 a_{1}, a_{2}+a_{8}=2 a_{5}$, and\n$$\na_{i+1} \\in\\left\\{1+a_{i}, 2+a_{i}\\right\\}, i=1,2, \\cdots, 9 \\text {, }\n$$\n\nthen the number of such sequences is $\\qquad$ .",
        "answer": "80",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 122267,
        "difficulty": null
      },
      {
        "id": "ec97950776163c6a96f3f5b9905503440a4d1c2f46708a8f1137e1145e793cdd",
        "domain": "numina",
        "question": "Question 223, Find the largest positive integer $n$, such that there exists a set with $n$ elements, where the set contains exactly 1 element divisible by $n$, exactly 2 elements divisible by $\\mathrm{n}-1$, $\\cdots$, exactly $\\mathrm{n}-1$ elements divisible by 2, and $\\mathrm{n}$ elements divisible by 1.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 138245,
        "difficulty": null
      },
      {
        "id": "d11126237699a13ce7491b95e8d16e9fa6226950ccf4058d2bd697e8c63421b7",
        "domain": "numina",
        "question": "5. (6 points) If $a<\\frac{1}{\\frac{1}{2011}+\\frac{1}{2012}+\\frac{1}{2013}+\\frac{1}{2014}+\\frac{1}{2015}}<a+1$, then the natural number $a=$ $\\qquad$",
        "answer": "402",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119419,
        "difficulty": null
      },
      {
        "id": "d83e2471302626c7b67bfe3b9ad6955351d355b0564457a54757d78b2127b11b",
        "domain": "numina",
        "question": "8.4. On the Island of Liars and Knights, a circular arrangement is called correct if each person standing in the circle can say that among their two neighbors, there is a representative of their tribe. Once, 2019 natives formed a correct circular arrangement. A liar approached them and said: \"Now we can also form a correct circular arrangement together.\" How many knights could there have been in the initial arrangement?",
        "answer": "1346",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 15747,
        "difficulty": null
      },
      {
        "id": "7d4b6ffe7d49e78a3a3ab0a095801a3f1f71495700951d918fc53ab86356d7e8",
        "domain": "numina",
        "question": "[ Dirichlet's Principle (continued). ] [Partitions into pairs and groups; bijections]\n\nBefore a clairvoyant lies a deck of 36 cards face down (4 suits, 9 cards of each suit). He names the suit of the top card, after which the card is revealed to him. Then he names the suit of the next card, and so on. The task of the clairvoyant is to guess the suit as many times as possible. The backs of the cards are asymmetrical, and the clairvoyant can see which of the two positions the top card is in. The clairvoyant's assistant knows the order of the cards in the deck, cannot change it, but can arrange the back of each card in one way or another. Could the clairvoyant have agreed with the assistant, when the latter still did not know the order of the cards, to ensure guessing the suit of at least\n\na) 19 cards;\n\nb) 23 cards",
        "answer": "23",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56920,
        "difficulty": null
      },
      {
        "id": "bc2234e5a4fdcfe47d438c07841e4af58fda05b26e1001b89dabcf0d5a2a05b7",
        "domain": "numina",
        "question": "(4) The parabola $y^{2}=2 p x(p>0)$ has a focus $F$, and a directrix $l$. $A$ and $B$ are two moving points on the parabola, and they satisfy $\\angle A F B=\\frac{\\pi}{3}$. Let the midpoint of segment $A B$ be $M$, and the projection of $M$ on $l$ be $N$. Then the maximum value of $\\frac{|M N|}{|A B|}$ is $\\qquad$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122706,
        "difficulty": null
      },
      {
        "id": "42ca1e97815343c6a6827cab21db0f8ef24c84cddc5aceb2e15dd61b8690b084",
        "domain": "numina",
        "question": "9. A, B, C, and D are comparing their heights. Among them, the sum of the heights of two people is the same as the sum of the heights of the other two. The average height of A and B is 4 cm more than the average height of A and C. D is 10 cm taller than A, and the sum of the heights of B and C is 288 cm. A's height is $\\qquad$ cm.",
        "answer": "139",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97607,
        "difficulty": null
      },
      {
        "id": "9a08fb5b14e79f1ddbce02e8b4ec3c7277b90d3bf2f54144e81b1a2ea6c8eaa2",
        "domain": "numina",
        "question": "# Problem 4.\n\nGiven a square with a side length of 5 cm. On its four sides, the vertices of a second square are located; on the four sides of the second square, the vertices of a third square, and so on. For what smallest natural $n$ will the sum of the areas of the first $n$ squares be guaranteed to be greater than 49 cm $^{2}$?",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5902,
        "difficulty": null
      },
      {
        "id": "162d19adb90fd075a54f2270dfa7bd60a53eaba9d91211924d7f07d9f329e5a3",
        "domain": "numina",
        "question": "9. Let $F(x)=|f(x) \\cdot g(x)|$, where $f(x)=a x^{2}+b x+c, g(x)=c x^{2}+b x+a, x \\in[-1,1]$. For any $a, b, c$ satisfying $|f(x)| \\leqslant 1$. When $a, b, c$ vary, find the maximum value of $F(x)$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113599,
        "difficulty": null
      },
      {
        "id": "fe795cb705e479b3c64bbe1bd06efccde1a3c9e25f4450c75bb057a5dd457bc0",
        "domain": "numina",
        "question": "9.3. Vasya must write one digit on each face of several dice so that any ordered combination of three digits from 000 to 999 inclusive can be obtained by selecting some three different dice and placing them with the appropriate sides up in the correct order. At the same time, the digits 6 and 9 do not transform into each other when rotated 180 degrees. What is the minimum number of dice Vasya should use?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9918,
        "difficulty": null
      },
      {
        "id": "3672a9693367e53f56fa44cb10510ce91c3771784c536725859324c1e378e161",
        "domain": "numina",
        "question": "7. If $\\cot \\alpha=2, \\tan (\\alpha-\\beta)=-\\frac{2}{5}$, then the value of $\\tan (\\beta-2 \\alpha)$ is",
        "answer": "-\\frac{1}{12}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109073,
        "difficulty": null
      },
      {
        "id": "1d01a3b4ece5af5e67c4f3c486fbfdcacdda564a276875e4795c1d1ab7f153b1",
        "domain": "numina",
        "question": "97. The distance between locations A and B is 300 kilometers. A car was originally planned to travel from A to B in 6 hours. The car traveled half the distance and then stopped for 30 minutes due to an issue. If it is to arrive at B on schedule, the car should increase its speed by $\\qquad$ kilometers/hour for the second half of the journey.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135582,
        "difficulty": null
      },
      {
        "id": "ad59f1f420d134205e9685acb4eb0830011be94c3ae0bede1865fc74b6637a8a",
        "domain": "numina",
        "question": "14. There are 40 red points, 30 blue points, and 20 green points marked on a circle. The circle is divided into 90 arcs, and each arc is labeled with a number based on the colors of its endpoints: a red-blue arc is labeled 1, a red-green arc is labeled 2, a blue-green arc is labeled 3, and arcs with endpoints of the same color are labeled 0. Find the minimum and maximum values of the sum of all these numbers.",
        "answer": "6140",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 103574,
        "difficulty": null
      },
      {
        "id": "708d9839ecdb179904a82536d1cef60cbff6a05614a7eebbb08a15e046d83bff",
        "domain": "numina",
        "question": "[ Rectangles and squares. Properties and characteristics ]\n\nSide $A B$ of rectangle $A B C D$ is equal to 12, and side $A D$ is equal to 5. The diagonals of the rectangle intersect at point $E$. Find the ratio of the distance from point $E$ to the center of the circle inscribed in triangle $A E D$ to the distance from point $E$ to the center of the circle inscribed in triangle $D E C$.\n\n#",
        "answer": "\\frac{10}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63169,
        "difficulty": null
      },
      {
        "id": "a1565d716e29dc23d3cc68a50a0ebd57a962810c219f32e9d4862308e60f9629",
        "domain": "numina",
        "question": "*end Let $S=\\{1,2, \\cdots, 2005\\}$. If any set of $n$ pairwise coprime numbers in $S$ contains at least one prime number, find the minimum value of $n$.",
        "answer": "16",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 106466,
        "difficulty": null
      },
      {
        "id": "05a22740f5893ee4ff7ace89c2ddaee7d1239e68a2dc2fd1eff2b16490a25e2f",
        "domain": "numina",
        "question": "8-1. Two rectangles $8 \\times 10$ and $12 \\times 9$ are overlaid as shown in the figure. The area of the black part is 37. What is the area of the gray part? If necessary, round the answer to 0.01 or write the answer as a common fraction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7d113eccc6d3ed32fb40g-15.jpg?height=471&width=577&top_left_y=684&top_left_x=745)",
        "answer": "65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14081,
        "difficulty": null
      },
      {
        "id": "f7dced2a6c54497db568c552dc4bb866a3c3aee30d05c2b2b748631975c01737",
        "domain": "numina",
        "question": "Example 3 Let $a$ be an integer such that $x^{2}-x+a$ divides $x^{13}+x+90$. Find the value of $a$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103783,
        "difficulty": null
      },
      {
        "id": "823f3c173afc9b4f354bfd47d0b16561e3dba6dc26c34df8ad9c7ae822e035c1",
        "domain": "numina",
        "question": "..... ]\n\n[ Auxiliary area. The area helps to solve the problem ]\n[ Thales' theorem and the theorem of proportional segments ]\n\nIn a right triangle with legs of 6 and 8, a square is inscribed, sharing a right angle with the triangle. Find the side of the square.\n\n#",
        "answer": "\\frac{24}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57799,
        "difficulty": null
      },
      {
        "id": "31caa76cb008f904477580c6186bffcdf935fde398ca620209d2377512c2e52c",
        "domain": "numina",
        "question": "6. Find the minimum value of the expression $x^{2}+y^{2}+3 z^{2}+2 x y$ given that $x y+z=1$ and $x, y, z>0$.",
        "answer": "\\frac{8}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6041,
        "difficulty": null
      },
      {
        "id": "344025eb8e031e5fdd01fdb9a1371cc4afe142f143ada86e47b58b83a2d5259c",
        "domain": "numina",
        "question": "## Task A-1.5.\n\nHow many numbers at least need to be removed from the set $\\{1,2,3, \\ldots, 2020\\}$ so that the resulting set does not contain the product of any two of its distinct elements?",
        "answer": "44",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89615,
        "difficulty": null
      },
      {
        "id": "d69b66f370b1d5856e8e9823ce4bd91b1dd19602f22beb49e084836e81e188b2",
        "domain": "numina",
        "question": "7.2. Vasya left the village of Vasilki and walked to the village of Romashki. At the same time, Roma left the village of Romashki for the village of Vasilki. The distance between the villages is 36 km, Vasya's speed is 5 km/h, and Roma's speed is 4 km/h. At the same time as Vasya, Dima left Vasilki on a bicycle at a speed equal to the sum of the boys' speeds and headed towards Roma. When he met Roma, he greeted him and immediately turned back. Meeting Vasya, he greeted him and immediately turned back. And so he rode back and forth between the boys until they met. How many kilometers did the cyclist travel during this time?",
        "answer": "36",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16833,
        "difficulty": null
      },
      {
        "id": "85e3b275b1ef575ac6d4d3111e8618e47a0a928d05201b0ae9f43d70dd368c13",
        "domain": "numina",
        "question": "Problem T-7. The numbers from 1 to $2013^{2}$ are written row by row into a table consisting of $2013 \\times 2013$ cells. Afterwards, all columns and all rows containing at least one of the perfect squares $1,4,9, \\ldots, 2013^{2}$ are simultaneously deleted.\n\nHow many cells remain?",
        "answer": "825423",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 176831,
        "difficulty": null
      },
      {
        "id": "710e0b8788a68b861431da2db8e592177848fb66cf02bd59a9630b1c310315e9",
        "domain": "numina",
        "question": "10. Taking the vertex $C(0,1)$ of the ellipse $x^{2}+a^{2} y^{2}=a^{2}(a>1)$ as the right-angle vertex, construct an inscribed isosceles right-angled triangle $A B C$ in this ellipse. The maximum number of such triangles that can be constructed is $\\qquad$.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 94235,
        "difficulty": null
      },
      {
        "id": "2c86fd1dc00c4dc82a359564799e761cf3ea6d907b9045676954506635fa3ee9",
        "domain": "numina",
        "question": "1. In a computer program written in Turbo Pascal, the function $\\operatorname{Random}(x)$ is used, generating integer random numbers from 1 to $x$. What is the probability that when this function is executed, a number divisible by 5 will appear if $x=100?$",
        "answer": "0.2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 31650,
        "difficulty": null
      },
      {
        "id": "7087ae2b1e3d8dc64f7b58f2f3f616b71e8f30a415f8711a0f47c42e52e21040",
        "domain": "numina",
        "question": "11.43*. In a city, there are 10 streets parallel to each other, and 10 streets intersecting them at right angles. What is the minimum number of turns a closed bus route can have, passing through all intersections?",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 27536,
        "difficulty": null
      },
      {
        "id": "72b74e809a0b443f9f9a6f9763315162bc6c3a25227dd4a1a78137754c7a34bf",
        "domain": "numina",
        "question": "6. In a perfectly competitive market, the demand function for a certain good is $\\mathrm{Q}_{\\mathrm{d}}(\\mathrm{p})=150-\\mathrm{p}$, and the supply function for this good is: $\\mathrm{Q}_{\\mathrm{s}}(\\mathrm{p})=3 \\mathrm{p}-10$. As a result of a sharp increase in the number of consumers of this good, under all other equal conditions, the demand for it increased by a factor of $\\alpha$ at every possible price level. As a result, the price of the good increased by $25 \\%$. Find the value of $\\alpha$. (8 points)",
        "answer": "1.4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7425,
        "difficulty": null
      },
      {
        "id": "712f319b5665ca836e1b019996efc777b880f4c0a84ccf672bc44119e2b0463a",
        "domain": "numina",
        "question": "Find all odd integers $n \\geqslant 1$ such that $n$ divides $3^{n}+1$.\n\n untranslated text remains unchanged.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 80846,
        "difficulty": null
      },
      {
        "id": "068bb1ae56f0cefe4970f11bef69c4b49b353058eb2123cc5a3caf644e27d795",
        "domain": "numina",
        "question": "10. In $\\triangle A B C$, $A B=\\sqrt{2}, A C=\\sqrt{3}$, $\\angle B A C=30^{\\circ}$, and $P$ is any point in the plane of $\\triangle A B C$. Then the minimum value of $\\mu=\\overrightarrow{P A} \\cdot \\overrightarrow{P B}+\\overrightarrow{P B} \\cdot \\overrightarrow{P C}+\\overrightarrow{P C} \\cdot \\overrightarrow{P A}$ is . $\\qquad$",
        "answer": "\\frac{\\sqrt{2}}{2}-\\frac{5}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 128043,
        "difficulty": null
      },
      {
        "id": "68a368635af72bb53945974da208a4df48291d1aa571c8059623785fc246669c",
        "domain": "numina",
        "question": "Suppose that $A B C D$ is a square with side length 4 and that $0<k<4$. Let points $P, Q, R$, and $S$ be on $B C, C D, D A$, and $A P$, respectively, so that\n\n$$\n\\frac{B P}{P C}=\\frac{C Q}{Q D}=\\frac{D R}{R A}=\\frac{A S}{S P}=\\frac{k}{4-k}\n$$\n\nWhat is the value of $k$ which minimizes the area of quadrilateral $P Q R S$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_bceb232fd49f7f0368beg-2.jpg?height=330&width=333&top_left_y=2082&top_left_x=1357)\n\n## PART B\n\nFor each question in Part B, your solution must be well-organized and contain words of explanation or justification. Marks are awarded for completeness, clarity, and style of presentation. A correct solution, poorly presented, will not earn full marks.",
        "answer": "\\frac{8}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194026,
        "difficulty": null
      },
      {
        "id": "48835bfcea3d4e86e1fa19db4564904894f33bea1def428427dac074e5f8eb17",
        "domain": "numina",
        "question": "39. Find the smallest integer starting with the digit 7 that decreases threefold when this digit is moved to the end. Find all such numbers.",
        "answer": "7241379310344827586206896551",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 29311,
        "difficulty": null
      },
      {
        "id": "90b0bd54d63e287766ec4c27b6f998109adf389a604602fc8c035a0bd755a245",
        "domain": "numina",
        "question": "In a group of $n$ people, each one had a different ball. They performed a sequence of swaps; in each swap, two people swapped the ball they had at that moment. Each pair of people performed at least one swap. In the end each person had the ball he/she had at the start. Find the least possible number of swaps, if: $a$ ) $n=5$; b) $n=6$.",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 197518,
        "difficulty": null
      },
      {
        "id": "cb2fbb609be1e52678a65b3217a68fbe6a96c4c219f5fe8e5f783718cf3ab553",
        "domain": "numina",
        "question": "6・65 Let positive integers $a, b$ be such that $15a + 16b$ and $16a - 15b$ are both squares of positive integers. Find the smallest value that the smaller of these two squares can take.\n(37th International Mathematical Olympiad, 1996)",
        "answer": "481^2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 127245,
        "difficulty": null
      },
      {
        "id": "1f166b2e3403ed07b3ff32529a762b71a0e5bd7baff61d06106869c06b0b2f74",
        "domain": "numina",
        "question": "16. 7 (US MO 16) In the plane, there are three circles $C_{i}(i=1,2,3)$, where the diameter of $C_{1}$ is $A B=1$; $C_{2}$ is concentric with $C_{1}$, has a diameter of $k$, and satisfies $1<k<3$; $C_{3}$ has $A$ as its center and $2 k$ as its diameter ($k$ is a constant). Consider all line segments $X Y$, one end $X$ on $C_{2}$, the other end $Y$ on $C_{3}$, and $X Y$ contains point $B$. For what ratio $X B: B Y$ is the length of segment $X Y$ minimized?",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122071,
        "difficulty": null
      },
      {
        "id": "6b3ac94aa79bf3c477da396b20e8427a27c5a7fb40fdb0ec3d73e938702a4895",
        "domain": "numina",
        "question": "5. The teacher invited a group of children to sit around a round table. There were three times as many boys as girls. She walked around the table and observed pairs of children sitting next to each other. She found that the number of pairs where both children were of the same gender was twice the number of pairs where the children were of different genders. What is the minimum number of children sitting around the table?\n\n## 53rd Mathematical Competition for High School Students in Slovenia\n\nSelection Competition, April 1, 2009\n\n## Problems for 4th Year",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 162892,
        "difficulty": null
      },
      {
        "id": "2c5f454a1d29e259e7f74e2ee7cc0dd4d0c104d8093ca71ce4be99d71acd3e44",
        "domain": "numina",
        "question": "1. (mathematics) There are scales with two pans, 4 weights of 2 kg each, 3 weights of 3 kg each, and two weights of 5 kg each. In how many different ways can a 12 kg load be balanced on the scales, if the weights are allowed to be placed on both pans?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 2820,
        "difficulty": null
      },
      {
        "id": "2e2b120bc280fb5ac3dd17cd411377c19779bb3aed121b9a90554cc5b5dc8f5f",
        "domain": "numina",
        "question": "3. (7 points) The number 20 is written on the board. In one move, it is allowed to either double the number or erase its last digit. Is it possible to get the number 25 in several moves?",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13770,
        "difficulty": null
      },
      {
        "id": "8a12aaa9462b0abb51bb1aaa4b828b728f2d1dffdb414a469105dac35618845d",
        "domain": "numina",
        "question": "3.1. Find the sum of all two-digit numbers for each of which the sum of the squares of the digits is 37 more than the product of the same digits.",
        "answer": "231",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4944,
        "difficulty": null
      },
      {
        "id": "56674d16aeec42d60287194b2d78be2332797120a423f5267e934cb0f2e101c0",
        "domain": "numina",
        "question": "[ $\\quad$ Number of divisors and their sum of a number [Fundamental Theorem of Arithmetic. Factorization into prime factors ]\n\na) Find the number $k$, which is divisible by 2 and 9 and has exactly 14 divisors (including 1 and $k$ ).\n\nb) Prove that if 14 is replaced by 15, the problem will have multiple solutions, and if 14 is replaced by 17, there will be no solutions at all.",
        "answer": "1458",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 71939,
        "difficulty": null
      },
      {
        "id": "2e04492dd0a20ae292a4b579e1611f1877ef343fe7ca897c9fa29176350bff9b",
        "domain": "numina",
        "question": "[ Algebraic inequalities (miscellaneous).] [ Case analysis $]$\n\n$x, y>0$. Let $S$ denote the smallest of the numbers $x, 1 / y, y+1 / x$. What is the maximum value that $S$ can take?",
        "answer": "\\sqrt{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 54345,
        "difficulty": null
      },
      {
        "id": "f28da5eca024dc87995fc6a5da2355baad61416d141e66999ee5c816d25e7a79",
        "domain": "numina",
        "question": "208. Random variables $X$ and $Y$ are independent. Find the variance of the random variable $Z=3X+2Y$, given that $D(X)=5, D(Y)=6$.",
        "answer": "69",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32349,
        "difficulty": null
      },
      {
        "id": "ab28b974598215d33ecb709debad86754db64036ec3f44a4fd166e15e58f1644",
        "domain": "numina",
        "question": "11. (20 points) Let 2018 real numbers $a_{1}, a_{2}, \\cdots, a_{2018}$ satisfy\n$$\n\\sum_{i=1}^{2018} a_{i}=0, \\sum_{i=1}^{2018} a_{i}^{2}=2018 .\n$$\n\nFind the maximum possible value of the minimum product of any two of these 2018 real numbers.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146870,
        "difficulty": null
      },
      {
        "id": "a369ae7f83619768b8dfc5a1b9769375c70e61cbb9e134291fddef68e0ad7c6f",
        "domain": "numina",
        "question": "8. (10 points) Given $n!=1 \\times 2 \\times 3 \\times \\cdots \\times n$, what is the result of the expression $\\frac{(2014!-1)!\\times 2015!}{(2014!)!}$? $\\qquad$ .",
        "answer": "2015",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 141850,
        "difficulty": null
      },
      {
        "id": "ebcff5388f90bede1b4e17e5dbb297b5da9ea226de0a57a83480e907a1ddf1b2",
        "domain": "numina",
        "question": "8. Given $f(x)=\\sum_{k=1}^{2017} \\frac{\\cos k x}{\\cos ^{k} x}$. Then $f\\left(\\frac{\\pi}{2018}\\right)=$",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115650,
        "difficulty": null
      },
      {
        "id": "31b02b8ed84296b124b7ec9a91e0ce54dbd947297e090175a159ac6a3621c3b5",
        "domain": "numina",
        "question": "Problem 8.2. In the chat of students from one of the schools, a vote was held: \"On which day to hold the disco: October 22 or October 29?\"\n\nThe graph shows how the votes were distributed an hour after the start of the voting.\n\nThen, 80 more people participated in the voting, voting only for October 22. After that, the voting ended. The final distribution of votes is also shown on the graph.\n\nHow many people participated in the voting?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0323bbf84409a1adeb34g-24.jpg?height=494&width=747&top_left_y=1295&top_left_x=339)",
        "answer": "260",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18592,
        "difficulty": null
      },
      {
        "id": "96a41e41d7cc28efa841e0eb3824a364de465fbed5c8a07768921dc23a3ed80e",
        "domain": "numina",
        "question": "A caravan of camels, 1 km long, moves uniformly across the desert. A rider rode from the end of the caravan to the beginning and returned to the end of the caravan. During this time, the caravan traveled 1 km. What distance did the rider travel if his speed was constant?",
        "answer": "1+\\sqrt{2}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 64058,
        "difficulty": null
      },
      {
        "id": "2a1e26a367a230be108568d57ff2bdf6fe8e2b11b5e533e6a370ef55fe4a8800",
        "domain": "numina",
        "question": "5. Determine the unknown number $x$ and explain.\n\n| 9 | 22 | 13 | 16 |\n| :---: | :---: | :---: | :---: |\n| 144 | 176 | 143 | 192 |\n| 16 | 8 | 11 | $x$ |\n\nTasks worth 10 points",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86853,
        "difficulty": null
      },
      {
        "id": "1aa94f25843608c0cb0f3f527aa6027ca8906ea5bf92a9e4cf536d48bf40bdca",
        "domain": "numina",
        "question": "2. (2 points) Find the minimum value of the expression $x^{2}-6 x \\sin y-9 \\cos ^{2} y$.",
        "answer": "-9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 20520,
        "difficulty": null
      },
      {
        "id": "f3f518c492914ccbbeb50f0df40e71d1ef9c13c4773b398fd25edd2b6baf8899",
        "domain": "numina",
        "question": "Problem 4. The bases $AB$ and $CD$ of trapezoid $ABCD$ are equal to 65 and 31, respectively, and its diagonals are perpendicular to each other. Find the scalar product of vectors $\\overrightarrow{AD}$ and $\\overrightarrow{BC}$.",
        "answer": "2015",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9065,
        "difficulty": null
      },
      {
        "id": "f6827d60be3c6960064048096bf1b49028815a6100c822ed2eceacf636050a9b",
        "domain": "numina",
        "question": "[\n\nA jeweler was commissioned to make a gold ring of width $h$, having the shape of a body bounded by the surface of a sphere with center $O$ and the surface of a cylinder of radius $r$, the axis of which passes through point $O$. The master made such a ring, but chose $r$ too small. How much gold will he have to add if $r$ needs to be increased by $k$ times, while keeping the width $h$ the same?\n\n#",
        "answer": "\\pi/6^{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64682,
        "difficulty": null
      },
      {
        "id": "157079ae85b543a6f5b90e26501f3523315217fc7400829de2fa3a36f4092616",
        "domain": "numina",
        "question": "Six bags contain 18, 19, 21, 23, 25, and 34 coins respectively, one of which is entirely filled with fake coins, while the other five bags do not contain any fake coins. Xiao Cong took three bags, and Xiao Min took the other two bags, leaving the bag with the fake coins. If the total number of coins Xiao Cong got is twice that of Xiao Min, then how many fake coins are there?",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114798,
        "difficulty": null
      },
      {
        "id": "aba30b49a519966cbb3bf610db7f94d353b67cb4652bb0d640b31c172533b805",
        "domain": "numina",
        "question": "| $A$ | $B$ | $C$ | $D$ | $E$ |\n| :--- | :--- | :--- | :--- | :--- |\n| $E$ | $D$ | $C$ | $B$ | $A$ |\n| $F$ | $F$ | $F$ | $F$ | $F$ |$+$\n\nA fragment of a conversation on the beach:\n\nFeri: Don't rush me now, I need to write digits here instead of the letters (Fig. 1), so that the addition is correct, the same digit for the same letter, and different digits for different letters. - Gyurka (after looking at the problem): What's the fuss about this? I can write down 77 solutions for you. - F: I doubt that. - Gy: I'll buy you as many ice creams as the number of solutions you miss, just come and swim now. - F: I'll go, but is the bet on? - How did the bet turn out?",
        "answer": "32",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 165621,
        "difficulty": null
      },
      {
        "id": "8fb214968ca85a458cbbd9b9ab50c8e0e268705b67a531239eff2a755e6733cb",
        "domain": "numina",
        "question": "1. Let $f(x)$ be a function defined on $R$, for any real number $x$ we have $f(x+3) \\cdot f(x-4)=-1$. Also, when $0 \\leq x<7$, $f(x)=\\log _{2}(9-x)$, then the value of $f(-100)$ is . $\\qquad$",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124190,
        "difficulty": null
      },
      {
        "id": "9a9f40189c09303bca98ed3c152c1906cd5ae6a7580520bbd43d6f065d820649",
        "domain": "numina",
        "question": "3. A three-stage launch vehicle consists of stages in the form of cylinders. All these cylinders are similar to each other. The length of the middle stage is two times less than the sum of the lengths of the first and third stages. In the fueled state, the mass of the middle stage is $13 / 6$ times less than the total mass of the fueled first and third stages. Find the ratio of the lengths of the first and third stages. The masses of the engines, instruments, and the stages' shells can be neglected.",
        "answer": "\\frac{7}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11329,
        "difficulty": null
      },
      {
        "id": "0b43c76882645fee3c7a958ac9aaf7a751dd944b8af348957913a3e91c0baa96",
        "domain": "numina",
        "question": "# 7. Variant 1\n\nUnit cubes were used to assemble a large parallelepiped with sides greater than 4. Two cubes will be called adjacent if they touch by faces. Thus, one cube can have up to 6 neighbors. It is known that the number of cubes that have exactly 6 neighbors is 836. Find the number of cubes that have no more than four neighbors.",
        "answer": "144",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18537,
        "difficulty": null
      },
      {
        "id": "97bc1e6ff557761f5ae8054d22f018326455545b3426c388c287cde5cbecf70a",
        "domain": "numina",
        "question": "2. The general solution of the equation $\\cos \\frac{x}{4}-\\cos x$ is ( ), within $(0,24 \\pi)$, there are ( ) distinct solutions.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135047,
        "difficulty": null
      },
      {
        "id": "2889fb4f3ffd2ade6e509544df009f29c05711f9b087bd581c74bde07ab019e8",
        "domain": "numina",
        "question": "14. Four problems were attempted by 100 contestants in a Mathematics competition. The first problem was solved by 90 contestants, the second by 85 contestants, the third by 80 contestants and the fourth by 75 contestants. What is the smallest possible number of contestants who solved all four problems?",
        "answer": "30",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 156404,
        "difficulty": null
      },
      {
        "id": "15229265c8a7beac2dc266632053710f3d36ae3b1b565a1b591b16dca006ff31",
        "domain": "numina",
        "question": "7. In a $4 \\times 4$ grid of 16 small squares, fill in 2 $a$s and 2 $b$s, with at most one letter in each small square. If the same letter is neither in the same row nor in the same column, there are $\\qquad$ ways to arrange them.",
        "answer": "3960",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143882,
        "difficulty": null
      },
      {
        "id": "fc34a198b0b1b0d4cac06e2b36688649b29ac5f100bfc0385e579d52fe037ed1",
        "domain": "numina",
        "question": "5. Dima has 25 identical bricks of size $5 \\times 14 \\times 17$. Dima wants to build a tower from all his bricks, each time adding one more brick on top (each new brick adds 5, 14, or 17 to the current height of the tower). We will call a number $n$ constructible if Dima can build a tower of height exactly $n$. How many constructible numbers exist?",
        "answer": "98",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4533,
        "difficulty": null
      },
      {
        "id": "3012b8ce09fe6b82b7e06da18a10407f872cac8317f6e9794fed70f482514ab3",
        "domain": "numina",
        "question": "A $5 \\times 5$ chessboard consists of unit squares, 7 of which are red and 18 are blue. Two of the red squares are located on the edge of the board. The segments that separate two adjacent red squares are also colored red. The segments that separate two adjacent blue squares are colored blue. All other segments, including the edges of the board, are black. In this way, a total of 35 black segments are formed. How many red segments are there?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 169363,
        "difficulty": null
      },
      {
        "id": "63fb1fda756373df86ab1ffe9cee393a63ca6f6e35c1a85ee5b79864c0d00485",
        "domain": "numina",
        "question": "4. Dina has a calculating machine, labelled $f$, that takes one number as input and calculates an output. The machine $f$ calculates its output by multiplying its input by 2 and then subtracting 3. For example, if Dina inputs 2.16 into $f$, the output is 1.32. If Dina inputs a number $x$ into $f$, she gets a first output which she then inputs back into $f$ to obtain a second output, which is -35 . What is the value of $x$ ?",
        "answer": "-\\frac{13}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138753,
        "difficulty": null
      },
      {
        "id": "38159305bba2d2d1f641eecd67875811fbd9377250b029ad485e56d0a2ed73dd",
        "domain": "numina",
        "question": "G6.1 If $\\log _{2} a-2 \\log _{a} 2=1$, find $a$.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152713,
        "difficulty": null
      },
      {
        "id": "befd90e6485993f069aabb9bd4bc12bb4b487d59b40bbb4ec3e1dd93d44e44db",
        "domain": "numina",
        "question": "The center of a circle touching the legs $A C$ and $B C$ of a right triangle $A B C$ lies on the hypotenuse $A B$. Find the radius of the circle, if it is six times smaller than the sum of the legs, and the area of triangle $A B C$ is 27.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66515,
        "difficulty": null
      },
      {
        "id": "bafd7dd3f6f3bb69748c2317eabab1dd79411497fd4e53f00d65f0b3fd76bc93",
        "domain": "numina",
        "question": "Example 4.34. Compute the integral $\\int_{0}^{1} \\frac{\\sin t}{t} d t$ with accuracy $\\varepsilon=10^{-4}$.",
        "answer": "0.9444",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31000,
        "difficulty": null
      },
      {
        "id": "7c50343b2c33f94d18a6d4516081fa4edd3364d6b93438b91ae755df6abe40ff",
        "domain": "numina",
        "question": "Consider 2018 lines in the plane, no two of which are parallel and no three of which are concurrent. Let $E$ be the set of their intersection points. We want to assign a color to each point in $E$ such that any two points on the same line, whose segment connecting them contains no other point of $E$, are of different colors.\n\nHow many colors are needed at a minimum to achieve such a coloring?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 84413,
        "difficulty": null
      },
      {
        "id": "745c5ab12dc27926e881c26de71c3e7db1c543c576f34e895696e999d8d51de9",
        "domain": "numina",
        "question": "【Question 13】There are 50 students in the third grade, all of whom choose to subscribe to one, two, or all three of the three magazines A, B, and C. Then at least $\\qquad$ students have the same subscription type.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117254,
        "difficulty": null
      },
      {
        "id": "23e21d5b0acbda103888258af02c41813062bdeb7554895d9afc3e63d96eafa6",
        "domain": "numina",
        "question": "XXVI - I - Problem 11\n\nA ship is moving at a constant speed along a straight line with an east-west direction. Every $ T $ minutes, the direction of movement is randomly chosen: with probability $ p $, the ship moves in the eastern direction for the next $ T $ minutes, and with probability $ q= 1-p $, it moves in the western direction. At a point outside the line, there is a submarine whose task is to torpedo the ship. The travel time of the torpedo from the point of firing to any point on the ship's track is $ 2T $. The captain of the submarine knows the value of $ p $ and aims to maximize the probability of hitting the ship. How should $ p $ be chosen to minimize the probability of the ship being torpedoed?",
        "answer": "\\frac{4}{9}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 842,
        "difficulty": null
      },
      {
        "id": "cb5f73890b4d31b96337fa29575b8b99ac0ca3513fe4dfff6f64877a4c80f63b",
        "domain": "numina",
        "question": "1. Find all real numbers $x$ for which\n\n$$\n|||| x|-2|-20|-200|=2007\n$$",
        "answer": "2229-2229",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162730,
        "difficulty": null
      },
      {
        "id": "d3548fa4e0ef2755b035cf0081f77d568b178663e66994056c564d23a95f3084",
        "domain": "numina",
        "question": "9. (6 points) If $\\overline{\\mathrm{ab}} \\times 65=\\overline{48 \\mathrm{ab}}$, then $\\overline{\\mathrm{ab}}=$",
        "answer": "75",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 98355,
        "difficulty": null
      },
      {
        "id": "ddc3e46a1e9fb0039f14b15d4846f3f5bcf387bd7f2be7c61fc1c2c71c1f647d",
        "domain": "numina",
        "question": "[Perpendicular Planes]\n\nThe base of the pyramid $M B K H E$ is a convex quadrilateral $M B K H$, where the angle at vertex $M$ is $\\pi / 2$, the angle formed by the diagonal $B H$ and the edge $B K$ is $\\pi / 4$, the length of the edge $M B$ is 1. The area of triangle $B K H$ is twice the area of triangle $M B H$. The sum of the lengths of edges $B E$ and $H E$ is $\\sqrt{3}$. The volume of the pyramid is 1/4. Find the radius of the sphere with the largest volume among all spheres that can fit inside the pyramid $M B K H E$.",
        "answer": "2-\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66021,
        "difficulty": null
      },
      {
        "id": "5bf977d1014ae004de4eeba9b7233e2851d8ccc9ece18bbb3337ae438b2964c1",
        "domain": "numina",
        "question": "2. You have a list of real numbers, whose sum is 40 . If you replace every number $x$ on the list by $1-x$, the sum of the new numbers will be 20 . If instead you had replaced every number $x$ by $1+x$, what would the sum then be?",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152196,
        "difficulty": null
      },
      {
        "id": "b6d3c536f71de4e5386617091bb5e922c42866a9b04bbb8682f0c1ae315c9fab",
        "domain": "numina",
        "question": "7. Given real numbers $a, b, c$ satisfy\n$$\n\\begin{array}{l}\na b c \\neq 0, a+b+c=a^{2}+b^{2}+c^{2}=2 . \\\\\n\\text { Then } \\frac{(1-a)^{2}}{b c}+\\frac{(1-b)^{2}}{c a}+\\frac{(1-c)^{2}}{a b}=\n\\end{array}\n$$\n$\\qquad$",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114047,
        "difficulty": null
      },
      {
        "id": "674009b49cdfc150f6b5671ea4353251a6c05b6411d80e6ac5b7aaa54d569573",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 1}\\left(\\frac{2 x-1}{x}\\right)^{\\frac{\\ln (3+2 x)}{\\ln (2-x)}}$",
        "answer": "\\frac{1}{5}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46521,
        "difficulty": null
      },
      {
        "id": "b19d5a944cc86300d357edfa8e9a1d7b8b4f4ebdce6e4040c3bd6bd044289d23",
        "domain": "numina",
        "question": "## Task 24/85\n\nLet there be a rectangle $A B C D$ with an internal point $P$. Determine $P A$ in terms of $P B, P C$, and $P D$.\n\nWhat value is obtained for $P B=33 \\mathrm{LE}, P C=28 \\mathrm{LE}, P D=41 \\mathrm{LE}$?",
        "answer": "\\sqrt{1996}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 182767,
        "difficulty": null
      },
      {
        "id": "069020134d98c132329d1d22dd55c03ee68108c78aa3dce151430699f8a8d582",
        "domain": "numina",
        "question": "## Task 2 - 040712\n\nA freight train traveled $35 \\frac{3}{4} \\mathrm{~km}$ in the first hour and in the following $2 \\frac{1}{2}$ hours, it covered an additional 92.7 $\\mathrm{km}$. For the return trip on the same route, it took three hours and 12 minutes. Calculate the average speed for the entire trip! Round to one decimal place!",
        "answer": "38.3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183510,
        "difficulty": null
      },
      {
        "id": "45cd109e50c591e0198ffd0df7a083812f6896935a8bfc83b09df7b28afe8514",
        "domain": "numina",
        "question": "[Example 2.2.1] Simplify: $M=\\frac{2}{\\sqrt{4-3 \\sqrt[4]{5}+2 \\sqrt[4]{25}-\\sqrt[4]{125}}}$.",
        "answer": "\\sqrt[4]{5}+1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140814,
        "difficulty": null
      },
      {
        "id": "01043d413def976bae87d909afb60ce865735ccb878d1714b2529520fea08593",
        "domain": "numina",
        "question": "A two-digit number has 3 as its tens digit, and when a digit is inserted in the middle to form a three-digit number, this three-digit number is exactly 9 times the original two-digit number. The new three-digit number is $\\qquad$",
        "answer": "315",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100263,
        "difficulty": null
      },
      {
        "id": "3e5acf5f82d74c6fdb7b4773d4c28d08ec0ef052e98ee70b4e46ddae8ce0c611",
        "domain": "numina",
        "question": "Problem 8.3. In the grove, there are 140 chameleons - blue and red. One day, several blue chameleons changed their color to red. As a result, the number of blue chameleons decreased by 5 times, and the number of red chameleons increased by 3 times. How many chameleons changed their color?",
        "answer": "80",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12110,
        "difficulty": null
      },
      {
        "id": "e7f1d6af5a97c442c564be8106db72c8a56d657bfb10e89973f2a35348989985",
        "domain": "numina",
        "question": "8. (10 points) On the right is an equation, where 9 Chinese characters represent the numbers 1 to 9, and different characters represent different numbers. The maximum possible value of the equation is $\\qquad$.\n\nHope $\\times$ Longing + Tree $\\times$ Green + Sky $\\times$ Blue",
        "answer": "8569",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 141414,
        "difficulty": null
      },
      {
        "id": "0c833886683d8eb04eb4fcbd004d8a9c77e95f942650875b1ff887681fb7fe5b",
        "domain": "numina",
        "question": "$4 \\cdot 64$ Find the product of the real roots of the equation $x^{2}+18 x+30=2 \\sqrt{x^{2}+18 x+45}$.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145739,
        "difficulty": null
      },
      {
        "id": "3b02a666a93a381483a138c1771ac1ef8fa0a914285315f5cb254f18a490bdd2",
        "domain": "numina",
        "question": "6. For any positive integers $x, y, z$, if $x \\mid y^{3}$, $y \\mid z^{3}$, $z \\mid x^{3}$, and $x y z \\mid (x+y+z)^{n}$, find the smallest positive integer $n$ that satisfies the condition.",
        "answer": "13",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136881,
        "difficulty": null
      },
      {
        "id": "abafb7fa84f463b4c68fc4eef00245d95eac14c7de7d60bdeea0edb0d271fdaf",
        "domain": "numina",
        "question": "21st BMO 1985 Problem 3 Let { x } denote the nearest integer to x, so that x - 1/2 ≤ { x } < x + 1/2. Define the sequence u 1 , u 2 , u 3 , ... by u 1 = 1. u n+1 = u n + { u n √2 }. So, for example, u 2 = 2, u 3 = 5, u 4 = 12. Find the units digit of u 1985 .",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 149170,
        "difficulty": null
      },
      {
        "id": "2fb86fe2d040ab4264cfc734855e694a7ca27c4a4e2a42687490f9899cc38642",
        "domain": "numina",
        "question": "2. Vasya and Petya simultaneously ran out from the starting point of a circular running track and ran in opposite directions at constant speeds. At some point, they met. Vasya ran a full circle and, continuing to run in the same direction, reached the place of their first meeting at the moment when Petya ran a full circle. Find the ratio of Vasya's and Petya's speeds.",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5100,
        "difficulty": null
      },
      {
        "id": "072aea08ba26ef3224ac97a2f898458193ddceff2a4cf40f599ddb3995a1fec1",
        "domain": "numina",
        "question": "5. Given that $x, y, z$ are 3 real numbers greater than or equal to 1, then\n$$\n\\left(\\frac{\\sqrt{x^{2}(y-1)^{2}+y^{2}}}{x y}+\\frac{\\sqrt{y^{2}(z-1)^{2}+z^{2}}}{y z}+\\frac{\\sqrt{z^{2}(x-1)^{2}+x^{2}}}{z x}\\right)^{2}\n$$\n\nthe sum of the numerator and denominator of the minimum value written as a simplified fraction is $\\qquad$ .",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107586,
        "difficulty": null
      },
      {
        "id": "37a32b603cf1a946b37fd7fcf23da91b8af735ed5101aab73e2b8ca6599f200c",
        "domain": "numina",
        "question": "6. In a rectangle $A B C D$ (including the boundary) with an area of 1, there are 5 points, among which no three points are collinear. Find the minimum number of triangles, with these 5 points as vertices, whose area is not greater than $\\frac{1}{4}$.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127518,
        "difficulty": null
      },
      {
        "id": "c233b7d6f1cc8ea7885babc8acccdd6acfab0674c32f1f953c2f7c86a1681080",
        "domain": "numina",
        "question": "3. Point $M$ is the midpoint of side $AB$ of triangle $ABC$. A circle $\\omega_{1}$ is drawn through points $A$ and $M$, tangent to line $AC$, and a circle $\\omega_{2}$ is drawn through points $B$ and $M$, tangent to line $BC$. Circles $\\omega_{1}$ and $\\omega_{2}$ intersect again at point $D$. Point $E$ lies inside triangle $ABC$ and is symmetric to point $D$ with respect to line $AB$. Find the angle $CEM$.",
        "answer": "180",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8215,
        "difficulty": null
      },
      {
        "id": "bc220715b31a3d2eadba2edf07fc9b7d5595f8600c51a91a2e30c14977fc0bae",
        "domain": "numina",
        "question": "In a board game played with a die, our pawn is four spaces away from the finish. If we roll at least a four, we reach the finish. If we roll a three, we will definitely be in with the next roll.\n\nWhat is the probability that we will reach the finish in only two more rolls?",
        "answer": "\\frac{1}{12}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 170449,
        "difficulty": null
      },
      {
        "id": "e3984207f384b58b4c80cb40d60f9919b79793460ae6d858fc5b12d0d251f737",
        "domain": "numina",
        "question": "15. In a $1 \\times 5$ grid, there are four chips, each with one side white and the other side black. Each operation allows you to select a chip to jump over one, two, or three chips to an empty position, but not to move it directly. The chips that are jumped over must be flipped, but the jumping chip remains the same. Now, it is desired to transform the situation on the left into the situation on the right in six operations. If the positions of the chips before they jump are recorded in sequence, a six-digit number can be formed. Please provide a possible six-digit number that can accomplish the task. $\\qquad$ (Provide one possible answer only).",
        "answer": "251425",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 106683,
        "difficulty": null
      },
      {
        "id": "70027ef5fde26fd214145c014657f74e916215029ce1c296d594b5aaafdf6edc",
        "domain": "numina",
        "question": "62. Use the six digits $2,3,4,5,6,7$ to form two three-digit numbers, so that the greatest common divisor (GCD) of these two three-digit numbers and 540 is as large as possible. The largest possible GCD is $\\qquad$.",
        "answer": "108",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103427,
        "difficulty": null
      },
      {
        "id": "f45d94b69e28f452e1bf50e2592b63b0f1b6f1f95c14e5d6b4998cc48276a480",
        "domain": "numina",
        "question": "KOVSHIN=BOTTLE+GLASS; $\\quad$ TWO KOVSHINS=SEVEN GLASSES; BOTTLE=CUP+TWO GLASSES; $\\quad$ BOTTLE=how many CUPS?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 71151,
        "difficulty": null
      },
      {
        "id": "c9b65a3cd6e04b4d7f286d039149423675cf356b286b2a99d1035bfb499547bb",
        "domain": "numina",
        "question": "Problem 8.3. How many lines exist that pass through the point $(0,2019)$ and intersect the parabola $y=x^{2}$ at two points with integer coordinates on the $y$-axis?\n\nAnswer: 9.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4515,
        "difficulty": null
      },
      {
        "id": "cf964812158208371a596addff4a54437eb52273930d178a84896566f00dd808",
        "domain": "numina",
        "question": "8. (10 points) In the equation, $\\overline{\\mathrm{ABCD}}+\\overline{\\mathrm{EFG}}=2020$, different letters represent different digits. Therefore, $A+B+C+D+E+F+G=$ $\\qquad$ .",
        "answer": "31",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 99387,
        "difficulty": null
      },
      {
        "id": "3d135e8b8c2bfb37dbc557249e5260436671fc1e70c73543174cfd294298c10f",
        "domain": "numina",
        "question": "Problem 6.6. Ksyusha runs twice as fast as she walks (both speeds are constant).\n\nOn Tuesday, when she left home for school, she first walked, and then, when she realized she was running late, she started running. The distance she walked was twice the distance she ran. In the end, it took her exactly 30 minutes to get from home to school.\n\nOn Wednesday, Ksyusha left home even later, so she had to run twice the distance she walked. How many minutes did it take her to get from home to school on Wednesday?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12097,
        "difficulty": null
      },
      {
        "id": "fe3c9bb166a754d4e567b3fb391d119d6a336085f714fc9614066c01034e9274",
        "domain": "numina",
        "question": "Example 2-55 In the permutations of the numbers $1,2,3, \\cdots, 9$, find the number of derangements where even numbers are in their original positions, and the rest are not in their original positions.",
        "answer": "44",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96508,
        "difficulty": null
      },
      {
        "id": "c7cb000874a86cbf33aecf8788e0a8bd05b90c74d67146f64b144fac872f09af",
        "domain": "numina",
        "question": "Biyankov Yu.A.\n\nIn an isosceles triangle \\(ABC\\), a point \\(M\\) is marked on the lateral side \\(BC\\) such that the segment \\(MC\\) is equal to the height of the triangle dropped to this side, and a point \\(K\\) is marked on the lateral side \\(AB\\) such that the angle \\(KMC\\) is a right angle. Find the angle \\(ACK\\).",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66269,
        "difficulty": null
      },
      {
        "id": "348c2b333ff86262d53803e8d0b93cda2f189f167b1608c66eaf648c6e36957e",
        "domain": "numina",
        "question": "3. Calculate:\n\n$$\n\\int_{0}^{\\frac{\\pi}{2}} \\frac{\\sin ^{2} x+\\sin x}{\\sin x+\\cos x+1} d x\n$$",
        "answer": "\\frac{\\pi}{4}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 179676,
        "difficulty": null
      },
      {
        "id": "816510489201401755e9d17946e77ff54eee1f76875b1212817c2126424edb8a",
        "domain": "numina",
        "question": "We are creating an infinite table of integers with the following procedure: The first column contains the terms of an arithmetic sequence in ascending order, whose first term is 4 and the common difference is 3. For every $k \\geq 1$, the $k$-th row is the arithmetic sequence whose common difference is $2k+1$. Where can the 1994 and the $1995$ be found in this table?",
        "answer": "1995",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167998,
        "difficulty": null
      }
    ],
    "numina_validation": [
      {
        "id": "d82629e5ef773d8eeefd7fbf9ee6f9a58bbb843059f76594e71cc9e0b481cc76",
        "domain": "numina",
        "question": "5. How many even six-digit numbers exist, in the notation of which identical digits do not stand next to each other",
        "answer": "265721",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17456,
        "difficulty": null
      },
      {
        "id": "5fd33ccd6be688af2d446362a3ebe1ad3e0d65953b865c409c6ca8887569bbf7",
        "domain": "numina",
        "question": "SG. 2 Given that the equation $|x|-\\frac{4}{x}=\\frac{3|x|}{x}$ has $k$ distinct real root(s), find the value of $k$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153255,
        "difficulty": null
      },
      {
        "id": "31ec9da053d7b0fbb9644a1bf81eaff5723e41093194a824a77e2b6959c71800",
        "domain": "numina",
        "question": "4. (42nd IMO Shortlist) Let $\\triangle ABC$ be an acute-angled triangle. Construct isosceles triangles $\\triangle DAC$, $\\triangle EAB$, and $\\triangle FBC$ outside $\\triangle ABC$ such that $DA = DC$, $EA = EB$, $FB = FC$, $\\angle ADC = 2 \\angle BAC$, $\\angle BEA = 2 \\angle ABC$, and $\\angle CFB = 2 \\angle ACB$. Let $D'$ be the intersection of line $DB$ and $EF$, $E'$ be the intersection of line $EC$ and $DF$, and $F'$ be the intersection of line $FA$ and $DE$. Find the value of $\\frac{DB}{DD'} + \\frac{EC}{EE'} + \\frac{FA}{FF'}$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93514,
        "difficulty": null
      },
      {
        "id": "53a09d684331e3d8517a8d5f2e007d39916f64a614f23ba0c472aa4f10b737ec",
        "domain": "numina",
        "question": "6. Piglet ran down a moving escalator and counted 66 steps. Then he ran up the same escalator at the same speed relative to the escalator and counted 198 steps. How many steps would he have counted if he had gone down a stationary escalator?\n\nAnswer. 99.",
        "answer": "99",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 22983,
        "difficulty": null
      },
      {
        "id": "96177c2e991523c37c03d9a388809e1f260248286158521890bc9e2d46c8c65d",
        "domain": "numina",
        "question": "1. Given the quadratic trinomial $f(x)=a x^{2}-a x+1$. It is known that $|f(x)| \\leqslant 1$ for all $x \\in[0,1]$. What is the greatest value that $a$ can take?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8357,
        "difficulty": null
      },
      {
        "id": "7abf8cb7d4722505fb5017ba6e13b6087d334a46714dd9668cbbd08a87054aaa",
        "domain": "numina",
        "question": "15. In a room, there are 3 people (Xiao Wang, Xiao Zhang, Xiao Li) and 1 dog. Every morning, the 3 people get up and make some cookies so that they can eat them whenever they are hungry. One morning, Xiao Wang was the first to leave for work. Before leaving, he threw 1 cookie to the dog and took and ate the remaining $1 / 3$; Xiao Zhang was the second to leave for work. Before leaving, he threw 1 cookie to the dog and took and ate the remaining $1 / 3$; Xiao Li was the third to leave for work. Before leaving, he threw 1 cookie to the dog and took and ate the remaining $1 / 3$; In the evening, after all 3 people returned home, they threw 1 cookie to the dog and then divided and ate the remaining cookies. Throughout the process, none of the cookies needed to be broken. Therefore, the minimum number of cookies Xiao Wang ate is ____.",
        "answer": "33",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 98242,
        "difficulty": null
      },
      {
        "id": "2ba0276b1cc713ec0195b185311f9dc6adffa4b700e87b2d1b07e5a21d47d310",
        "domain": "numina",
        "question": "Example 11 Divide a circle with a circumference of 24 into 24 equal segments, and select 8 points from the 24 points such that the arc length between any two points is not equal to 3 or 8. How many different ways are there to select such a group of 8 points? Explain your reasoning.\n(2001 CMO Problem)",
        "answer": "258",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146154,
        "difficulty": null
      },
      {
        "id": "47cd26846efc9a4f9a817e798a5a551aa8ec009435e579c5c6607d7412556b6e",
        "domain": "numina",
        "question": "Calculate $\\frac{\\pi}{\\sqrt{2}}$ to three decimal places.",
        "answer": "2.221",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 173224,
        "difficulty": null
      },
      {
        "id": "4568016e757dde2d531366f2137cb23ce75fb5fc1a5dd5ce817570a501b44e31",
        "domain": "numina",
        "question": "## 29. Apples\n\nAmelie picked apples, ate a quarter of them herself, and gave the rest to her little sisters Bertha, Charlotte, and Dorothy. Bertha ate a quarter of her apples and gave the rest to her three sisters. Charlotte ate one apple and gave the rest to her three sisters. As for little Dorothy, she also ate a quarter of her share before giving the rest to her three sisters.\n\n\"Look,\" said Bertha to Charlotte, \"I have twice as many apples left as you do.\"\n\nHow many apples did Amelie pick?",
        "answer": "32",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 42208,
        "difficulty": null
      },
      {
        "id": "0fb0b17cbd3da3ffeff8034c07889cd03eacdebb9adb699dabcd660389f96fbe",
        "domain": "numina",
        "question": "Problem 1. The external angles of a triangle are $20\\%, 35\\%$ and $45\\%$ of the sum of its external angles. Determine the angle between the bisector of the smallest angle and the shortest side.",
        "answer": "53",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188543,
        "difficulty": null
      },
      {
        "id": "4d1b3dc8bf9b22368b2eea883f1d1294db130154c7a39f99863b93a1d17d40e5",
        "domain": "numina",
        "question": "11. As shown in the figure, in the Cartesian coordinate system, the ellipse $\\Gamma$ : $\\frac{x^{2}}{2}+y^{2}=1$ has left and right foci $F_{1}, F_{2}$, respectively. Let $P$ be a point on $\\Gamma$ in the first quadrant, and the extensions of $P F_{1}, P F_{2}$ intersect $\\Gamma$ at points $Q_{1}, Q_{2}$, respectively. Let $r_{1}, r_{2}$ be the inradii of $\\triangle P F_{1} Q_{2}, \\triangle P F_{2} Q_{1}$, respectively. Find the maximum value of $r_{1}-r_{2}$.",
        "answer": "\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126149,
        "difficulty": null
      },
      {
        "id": "17cff180c485e8aa1534707cb221ee6e88cf6fc6e43113a713c1dfc7083bd02a",
        "domain": "numina",
        "question": "If $\\frac{8}{24}=\\frac{4}{x+3}$, what is the value of $x$ ?",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 193968,
        "difficulty": null
      },
      {
        "id": "e12ccd1678216d3ae3d9a38052135d8b7e2522213b567dc9809ea8d060db16ba",
        "domain": "numina",
        "question": "8. (10 points) Max fills 9 different natural numbers into the nine blank spaces in the figure, so that the sum of the 3 numbers in each row, each column, and each diagonal is equal. It is known that the difference between $A$ and $B$ is $14$, and the difference between $B$ and $C$ is also $14$. Therefore, the difference between $D$ and $E$ is . $\\qquad$",
        "answer": "49",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 129599,
        "difficulty": null
      },
      {
        "id": "f0691b0dc0d1913ce03ab061707c929fcb79a52a7cbcbaa29bbcd7248388f6e1",
        "domain": "numina",
        "question": "In certain cassette tapes, a $0,0075 \\mathrm{~mm}$ thick tape is wound on spools with a diameter of $11 \\mathrm{~mm}$. The distance between the centers of the spools is $42 \\mathrm{~mm}$. What is the maximum length of tape that can be in a usable cassette? (From the full spool, the tape must pass freely to the empty spool.)",
        "answer": "344",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 169295,
        "difficulty": null
      },
      {
        "id": "d1f7e20c17e6c5cbdbdda08802772c8a5b21550312ce595b64c439780126d131",
        "domain": "numina",
        "question": "932. Find two natural numbers if their product is between 120 and 130, and their ratio is between 2 and 3. List all solutions.",
        "answer": "187",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 45322,
        "difficulty": null
      },
      {
        "id": "b8ef3aa142fa83cd560b6d6cdd747fdf9e20d3844e37103ce96f0cd2060e574f",
        "domain": "numina",
        "question": "4.9. Two equal triangles $K L M$ and $K L N$ share a common side $K L$, and $\\angle K L M=\\angle L K N=60^{\\circ}$, $K L=1$ and $L M=K N=6$. The planes $K L M$ and $K L N$ are perpendicular. Find the radius of the sphere that touches the segments $L M$ and $K N$ at their midpoints.",
        "answer": "\\frac{137}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 24153,
        "difficulty": null
      },
      {
        "id": "02deb7bea59b0d169b07d34a2f2b702b592f5dcd6907e5ac9019b0619210b8fe",
        "domain": "numina",
        "question": "2. A barrel 1.5 meters high is completely filled with water and covered with a lid. The mass of the water in the barrel is 1000 kg. A long thin tube with a cross-sectional area of $1 \\mathrm{~cm}^{2}$ is inserted vertically into the lid of the barrel, which is completely filled with water. Find the length of the tube if it is known that after it is filled, the pressure at the bottom of the barrel has doubled. The density of water is 1000 kg $/ \\mathbf{M}^{3}$.",
        "answer": "1.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11375,
        "difficulty": null
      },
      {
        "id": "d651a867ba57b0aa401e20dc57827c2a9171377c92bd363fa47988aa8eed59df",
        "domain": "numina",
        "question": "7. If the expression $\\frac{1}{1 \\times 2}-\\frac{1}{3 \\times 4}+\\frac{1}{5 \\times 6}-\\frac{1}{7 \\times 8}+\\cdots-\\frac{1}{2007 \\times 2008}+\\frac{1}{2009 \\times 2010}$ is converted to a decimal, then the first digit after the decimal point is $\\qquad$ .",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124341,
        "difficulty": null
      },
      {
        "id": "0d88db6eda539033ab852a8e9ca59689837058d5a9c494454269c745eb5c961f",
        "domain": "numina",
        "question": "Task 2. (10 points) A circle touches the extensions of two sides $AB$ and $AD$ of square $ABCD$, and the point of tangency cuts off a segment of length 2 cm from vertex $A$. Two tangents are drawn from point $C$ to this circle. Find the side of the square if the angle between the tangents is $30^{\\circ}$, and it is known that $\\sin 15^{\\circ}=\\frac{\\sqrt{3}-1}{2 \\sqrt{2}}$.\n\n#",
        "answer": "2\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7206,
        "difficulty": null
      },
      {
        "id": "1384a5c2993ee9827fcb6ae5929738a6ac97eab067f751e4293f950965f62ff5",
        "domain": "numina",
        "question": "Opgave 1. Zij $f: \\mathbb{Z}_{>0} \\rightarrow \\mathbb{R}$ een functie waarvoor geldt: voor alle $n>1$ is er een priemdeler $p$ van $n$ zodat\n\n$$\nf(n)=f\\left(\\frac{n}{p}\\right)-f(p)\n$$\n\nBovendien is gegeven dat $f\\left(2^{2014}\\right)+f\\left(3^{2015}\\right)+f\\left(5^{2016}\\right)=2013$.\n\nBereken $f\\left(2014^{2}\\right)+f\\left(2015^{3}\\right)+f\\left(2016^{5}\\right)$.",
        "answer": "\\frac{49}{3}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85454,
        "difficulty": null
      },
      {
        "id": "f3c0b3dec3fa482064632cc09655ab5074b518fdc02197fdc7f4ec603540fcf5",
        "domain": "numina",
        "question": "$12.36 \\lim _{x \\rightarrow 1} \\frac{\\sqrt[3]{x}-1}{\\sqrt[3]{x^{2}+2 \\sqrt[3]{x}-3}}$.\n\nCalculate the limits, and then confirm or refute the following statements ( $12.37-12.38$ ):",
        "answer": "0.25",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 50006,
        "difficulty": null
      },
      {
        "id": "eb9adf4123de0aeb52158ade2e5367d4288b2521c47d4815b9f152388031aaf8",
        "domain": "numina",
        "question": "11. (40 points) For the quadratic function $p(x)=a x^{2}+b x+c$, for some integers $n$, the equality $p(n)=p\\left(n^{2}\\right)$ holds. Provide an example of the function $p(x)$ for which the number of such numbers $n$ is the largest. What is this largest number of numbers $n$?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8252,
        "difficulty": null
      },
      {
        "id": "56981edf1fd0f6764bd3babace782997d44a33574f0d3402e97360eb18daaa6c",
        "domain": "numina",
        "question": "$2.20 \\sqrt[7]{\\frac{5-x}{x+3}}+\\sqrt[7]{\\frac{x+3}{5-x}}=2$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49513,
        "difficulty": null
      },
      {
        "id": "a801b45c658734dd789e9270562eb1577fb99aa8b83280a58352a8b4029ada73",
        "domain": "numina",
        "question": "3. On the sides $A B$ and $B C$ of an equilateral triangle $A B C$, points $P$ and $Q$ are chosen such that $A P: P B = B Q: Q C = 2: 1$. Find $\\angle A K B$, where $K$ is the intersection point of segments $A Q$ and $C P$.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7979,
        "difficulty": null
      },
      {
        "id": "6af7e034eea40c80c4b5b8fb65cfd20dda64352a6dbb6fd5f2159a666115a2bd",
        "domain": "numina",
        "question": "## Task A-4.7.\n\nA figure placed on the surface of a cube $K_{n}$ of dimensions $n \\times n \\times n$ on a side attacks all the fields in the row and column in which it is located, similar to a chess rook, but also the fields on the other sides in the extensions of those rows/columns. (In the figure, the visible fields on the cube $K_{4}$ that the placed figure attacks are marked.)\n\nHow many figures can we place on the surface of the cube $K_{50}$ so that they do not attack each other?\n\n![](https://cdn.mathpix.com/cropped/2024_05_30_ce71b02a7f068198719dg-28.jpg?height=323&width=303&top_left_y=2026&top_left_x=1385)",
        "answer": "75",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89099,
        "difficulty": null
      },
      {
        "id": "04008e3dd5b70f2f3175e3a9dbcd0c62f19d1820a70819b2b0eda71772aea314",
        "domain": "numina",
        "question": "## Task B-4.5.\n\nAndro and Borna take turns rolling a fair die whose sides are marked with the numbers 1, 2, 3, 4, 5, and 6. The winner is the first one to roll a six. If Andro starts the game, what is the probability that Borna wins?",
        "answer": "\\frac{5}{11}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88196,
        "difficulty": null
      },
      {
        "id": "873671e2178b0dfc3df6c18c809b431fee1399ea72174cedf29af201d53d55e3",
        "domain": "numina",
        "question": "3. Find the smallest natural number $n$, such that among any $n$ irrational numbers, there always exist 3 numbers, the sum of any two of which is still an irrational number.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 138760,
        "difficulty": null
      },
      {
        "id": "a72e805f47ac1ab6035d10e6bbbd38cb0b260bf6969d0c7c64e46b67a1e358e5",
        "domain": "numina",
        "question": "12. The maximum value of the function $S=\\sqrt{x^{4}-5 x^{2}-8 x+25}-\\sqrt{x^{4}-3 x^{2}+4}$ is",
        "answer": "\\sqrt{17}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109513,
        "difficulty": null
      },
      {
        "id": "fc8995326afaa3a97c5ed556ecc9022dd1962aecddbaacc61d693e223f472bf0",
        "domain": "numina",
        "question": "3 Let $x, y$ be real numbers, $2x + y \\geqslant 1$, then the minimum value of the bivariate function $u = x^{2} + 4x + y^{2} - 2y$ is $\\qquad$ .",
        "answer": "-\\frac{9}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145492,
        "difficulty": null
      },
      {
        "id": "b40eed7e4ead437e35b0aa0a9b604e13e3c1c2173b03a0ec8f56aea3027641ac",
        "domain": "numina",
        "question": "5. From $1,2,3, \\cdots, 10$, take three different numbers, where the sum of two numbers is twice the other number, there are $\\qquad$ different ways.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128283,
        "difficulty": null
      },
      {
        "id": "f4deb86c02ef68d7f8597475b5dee47c7ffd34d0a0cab016ddf311dc3b71586a",
        "domain": "numina",
        "question": "Agakhanovo $H . X$.\n\nDifferent numbers $a, b$ and $c$ are such that the equations $x^{2}+a x+1=0$ and $x^{2}+b x+c=0$ have a common real root. In addition, the equations $x^{2}+x+a=0$ and $x^{2}+c x+b=0$ have a common real root. Find the sum $a+b+c$.",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70967,
        "difficulty": null
      },
      {
        "id": "eb425d8a98adc60c841d6be4d28931e36fb90029b405ebc9ef24b6a1d71f9df4",
        "domain": "numina",
        "question": "Task 2. (10 points) A circle touches the extensions of two sides $A B$ and $A D$ of square $A B C D$ with side $\\sqrt{2+\\sqrt{2}}$ cm. Two tangents are drawn from point $C$ to this circle. Find the radius of the circle if the angle between the tangents is $45^{\\circ}$, and it is known that $\\sin 22.5^{\\circ}=\\frac{\\sqrt{2-\\sqrt{2}}}{2}$.\n\n#",
        "answer": "\\sqrt{2}+\\sqrt{2-\\sqrt{2}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7170,
        "difficulty": null
      },
      {
        "id": "2d05208001628b21c5a63583d1f6417f148896d82e507c3d084820036edcba27",
        "domain": "numina",
        "question": "At 12 o'clock, the three hands of the clock meet. The question is, how many minutes later will the second hand bisect the angle formed by the hour and minute hands?",
        "answer": "1\\frac{13}{1427}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 175327,
        "difficulty": null
      },
      {
        "id": "8402715e349a90ffd652c47f6ebd1354b63bb33de996525989babec8a2eacf42",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the volumes of the bodies bounded by the surfaces.\n\n$$\n\\frac{x^{2}}{9}+\\frac{y^{2}}{4}-z^{2}=1, z=0, z=4\n$$",
        "answer": "152\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47245,
        "difficulty": null
      },
      {
        "id": "aeb9c159cf233789661c796b415b8abc055e89e2cab92eb30eb913153839ba93",
        "domain": "numina",
        "question": "Anna writes a sequence of 0s and 1s on the board. Anna notices that in every sequence of 200 consecutive digits written on the board, there are as many 0s as there are 1s. She also notices that in every sequence of 202 consecutive digits written on the board, the number of 0s and 1s is not equal. What is the maximum number of digits that Anna could have written on the board?",
        "answer": "300",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 79777,
        "difficulty": null
      },
      {
        "id": "6db8fe3852a23844482756e8cfc836de9082e293d238740cb318077e8b6395c3",
        "domain": "numina",
        "question": "【Question 3】There is a basket of apples. The first time, take out half of all plus 2 more. The second time, take out half of the remaining minus 3. There are still 24 left in the basket. The basket originally had $\\qquad$ apples.",
        "answer": "88",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99362,
        "difficulty": null
      },
      {
        "id": "a34e8ee6e8cd9894ad288263ae83cd42ecb228b96a58d249af9b4076a194f0d9",
        "domain": "numina",
        "question": "Problem 3. A four-digit number $X$ is not divisible by 10. The sum of the number $X$ and the number obtained from $X$ by swapping its second and third digits is divisible by 900. Find the remainder when the number $X$ is divided by 90.",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9074,
        "difficulty": null
      },
      {
        "id": "12f85d9b9d28a28c41766d18feb62e9af210c72c4bc1decbdb743077932af2ad",
        "domain": "numina",
        "question": "Subject I\nDetermine the remainder of the division of the number $1 \\cdot 2 \\cdot 3 \\cdot 4 \\cdot \\ldots \\cdot 69 \\cdot 70 + 1234$ by 2013.",
        "answer": "1234",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 179118,
        "difficulty": null
      },
      {
        "id": "2eea45e3ba591185239d8a839006d81d7c0acbc75a7591b4447001c570371121",
        "domain": "numina",
        "question": "3. The faces of a hexahedron and the faces of a regular octahedron are all equilateral triangles with side length $a$. The ratio of the radii of the inscribed spheres of these two polyhedra is a reduced fraction $\\frac{m}{n}$. Then, the product $m \\cdot n$ is $\\qquad$.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93370,
        "difficulty": null
      },
      {
        "id": "3876db41d560c37e0b6abda8a25f9064d4b16a21f0e6af7376fa69031ec28504",
        "domain": "numina",
        "question": "2. Find all three-digit numbers with this property: if we strike out the middle digit of the number and multiply the resulting two-digit number by the square of the struck-out digit, we get the original three-digit number again.\n\n(Tomáš Jurík)",
        "answer": "135",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75893,
        "difficulty": null
      },
      {
        "id": "b6a17d544334353f8ec3e72400e03ca895ab5021b9d70a5ff3c0f5e10f0ae02c",
        "domain": "numina",
        "question": "3. A $10 x 10$ grid is filled with non-negative numbers. It is known that in each (vertical or horizontal) strip of 1 x 3, the sum of the numbers is 9. What is the maximum value that the sum of all numbers in the grid can take?",
        "answer": "306",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20976,
        "difficulty": null
      },
      {
        "id": "e7163a23c11037a1b28b5b3a4605fcb6594b24b9307a3a7a5700f614d10d15b8",
        "domain": "numina",
        "question": "For $n \\leqslant 100$, we have $f(n)=91$.\n(1983 Yugoslav Olympiad)",
        "answer": "91",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 102137,
        "difficulty": null
      },
      {
        "id": "df2837ca7e83b47ff420e4696a67ed03999bd3ae4e07aa8033dd2fa9721c6e31",
        "domain": "numina",
        "question": "6. As shown in the figure, in the tetrahedron $P-ABC$, $PA \\perp$ plane $ABC, \\angle ABC=120^{\\circ}, PA=4$. If the radius of the circumscribed sphere of the tetrahedron $P-ABC$ is $2 \\sqrt{2}$, then the tangent value of the angle formed by line $PC$ and plane $ABC$ is $\\qquad$.",
        "answer": "\\frac{2\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119283,
        "difficulty": null
      },
      {
        "id": "d1366cb229a98737ce4ea5a8a87e7543b33c7f5b0ce1e05f7e54cbf63cd2513a",
        "domain": "numina",
        "question": "20. From the 10 natural numbers 0 to 9, select 4 different numbers such that the sum of the selected 4 numbers is a multiple of 3. There are $\\qquad$ ways to do this.",
        "answer": "72",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 109935,
        "difficulty": null
      },
      {
        "id": "ae99241b80fae0975633f5545fe2cfd2d93118809af9491fc09db43a9050eb5e",
        "domain": "numina",
        "question": "14. Let $x, y, z$ be positive real numbers, find the minimum value of $\\left(x+\\frac{1}{y}+\\sqrt{2}\\right)\\left(y+\\frac{1}{z}+\\sqrt{2}\\right)\\left(z+\\frac{1}{x}+\\sqrt{2}\\right)$.",
        "answer": "20+14\\sqrt{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 119946,
        "difficulty": null
      },
      {
        "id": "4360fc8d312cd98d071af11b64728d42564d93c67f0a05f33ca3116fc2a536c9",
        "domain": "numina",
        "question": "39. Among the numbers $1,2,3, \\cdots \\cdots, 100$, the maximum number of numbers that can be selected such that the product of any two numbers is not 120 is $\\qquad$.",
        "answer": "93",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 106623,
        "difficulty": null
      },
      {
        "id": "b21696d18e7d506044b639fa03608f3d751735a9700c29af61f0b30617b627cd",
        "domain": "numina",
        "question": "12. (6 points) The natural number $a$ is a multiple of 3, $a-1$ is a multiple of 4, $a-2$ is a multiple of 5, then the smallest $a$ is $\\qquad$",
        "answer": "57",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 95180,
        "difficulty": null
      },
      {
        "id": "be5558678cd0caa9c36ff2e743d3cc5adbaad8f52f17e7068485762a4f679e96",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{1+4+7+\\ldots+(3 n-2)}{\\sqrt{5 n^{4}+n+1}}\n$$",
        "answer": "\\frac{3}{2\\sqrt{5}}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46466,
        "difficulty": null
      },
      {
        "id": "f368a6be1cecf698693e84122fbb33126a917f554b700d81d0d7c8ec116ef046",
        "domain": "numina",
        "question": "As shown in Figure 1, in $\\triangle A B C$, $A C-A B=\\frac{\\sqrt{2}}{2} B C$. Point $D$ is a point inside $\\triangle A B C$ such that $A D=A B$. A perpendicular line is drawn from point $D$ to $B C$, with the foot of the perpendicular being $E$. If $3 B E=C E$, find the measure of $\\angle A D E$.",
        "answer": "135",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 134122,
        "difficulty": null
      },
      {
        "id": "ba8183cc3d073c6287a4fe590824eadb45f30468724af7df23a2f9820a0ae562",
        "domain": "numina",
        "question": "4B. Solve the equation\n\n$$\n\\log _{3 x+4}(2 x+1)^{2}+\\log _{2 x+1}\\left(6 x^{2}+11 x+4\\right)=4\n$$",
        "answer": "\\frac{3}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190291,
        "difficulty": null
      },
      {
        "id": "edd7d08fa035eb4a4d7edd0df814f3607e4ee96feac90a4bad2b029c11f29c03",
        "domain": "numina",
        "question": "4. (10 points) Person A leaves location $A$ to find person B, and after walking 80 kilometers, arrives at location $B$. At this point, person B had left for location $C$ half an hour earlier. Person A has been away from location $A$ for 2 hours, so A continues to location $C$ at twice the original speed. After another 2 hours, both A and B arrive at location $C$ simultaneously. What is the speed of person B in kilometers per hour?",
        "answer": "64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134484,
        "difficulty": null
      },
      {
        "id": "571286b3efdee48d31297473364af77507398a321635c9fb230f23f66e3c7e25",
        "domain": "numina",
        "question": "$4 \\cdot 87$ Solve the equation $\\log _{x} 3 \\cdot \\log _{3 x} 3=\\log _{9 x} 3$.",
        "answer": "3^{\\\\sqrt{2}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101946,
        "difficulty": null
      },
      {
        "id": "ddf5c73b7eea0b4ac6a1a6fffa83092cc3f57835bfff75e66eab4a1139d4abf4",
        "domain": "numina",
        "question": "9. The sum of the lengths of all curve segments formed by a moving point on the surface of a regular quadrilateral pyramid $P-ABCD$ with lateral edge length and base edge length both being 4, and at a distance of 3 from the vertex $P$, is $\\qquad$ .",
        "answer": "6\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 139716,
        "difficulty": null
      },
      {
        "id": "dfac024f1167d9ba3e59272747edcd3bdd66565cf5e150bb2c6a13baa85b8abe",
        "domain": "numina",
        "question": "The ants Maricota and Nandinha are strolling on a balcony whose floor is made up of rectangular tiles measuring $4 \\mathrm{~cm}$ in width by $6 \\mathrm{~cm}$ in length. Maricota starts from point $M$ and Nandinha from $N$, both walking only along the sides of the rectangles, following the path indicated in the figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_b9d1398cc05f7e479435g-1.jpg?height=317&width=500&top_left_y=1252&top_left_x=869)\n\n(a) The two meet after walking the same distance. What was that distance?\n\n(b) Where did they meet?",
        "answer": "66",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74261,
        "difficulty": null
      },
      {
        "id": "4ac19b1e2c24e39004820e65c5ebac92e725bb05492be72ce980a9871c8e933d",
        "domain": "numina",
        "question": "1. How many four-digit numbers exist that are divisible by 17 and end in 17?",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14267,
        "difficulty": null
      },
      {
        "id": "f94c6c38aa4b8d056977734c977de778bbd50953ef7fc2f52e3f014ce493c5c9",
        "domain": "numina",
        "question": "3. (8 points) Rongrong transferred from Class 1 to Class 2, and Leilei transferred from Class 2 to Class 1. As a result, the average height of students in Class 1 increased by 2 cm, and the average height of students in Class 2 decreased by 3 cm. If Leilei's height is 158 cm and Rongrong's height is 140 cm, then the total number of students in the two classes is $\\qquad$ people.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115971,
        "difficulty": null
      },
      {
        "id": "695b1d931b75e4c1ce5590d5d6d072e0e40879ae5ed8281315bca1725b0f587c",
        "domain": "numina",
        "question": "1. On January 1, 2013, a little boy was given a bag of chocolate candies, containing 300 candies. Each day, the little boy ate one candy. On Sundays, Karlson would fly over, and the little boy would treat him to a couple of candies. How many candies did Karlson eat? (January 1, 2013, was a Tuesday).\n\n2. Petya can swap any two digits of different parity in the number 1974835. What is the largest number he can obtain this way?",
        "answer": "66",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13730,
        "difficulty": null
      },
      {
        "id": "df5f2bbb48a603cb02013187331425f57fdda010e18b5bf62646d8af98eac160",
        "domain": "numina",
        "question": "Four, (50 points) During the Teachers' Day at a certain school, 200 teachers sent text messages to greet each other, with each person sending exactly one message to another teacher. Now, from them, the maximum number of $k$ representatives can be selected to attend a commendation conference, such that none of them has sent a message to any of the others. Try to find the minimum value of $k$.\n\n---\n\nThe above text has been translated into English, preserving the original text's line breaks and format.",
        "answer": "67",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117612,
        "difficulty": null
      },
      {
        "id": "73c08e67e42e1096d2cedc70c58e6378a237c8f0a897960df45b541a18d21dcf",
        "domain": "numina",
        "question": "33. In the following diagram, $A B C D$ is a square, and $E$ is the center of the square $A B C D . P$ is a point on a semi-circle with diameter $A B$. $Q$ is a point on a semi-circle with diameter $A D$. Moreover, $Q, A$ and $P$ are collinear (that is, they are on the same line). Suppose $Q A=14$ $\\mathrm{cm}, A P=46 \\mathrm{~cm}$, and $A E=x \\mathrm{~cm}$. Find the value of $x$.",
        "answer": "34",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160246,
        "difficulty": null
      },
      {
        "id": "3c186fc383e954a2fc500cfe9b7b2b3f7aa5efbfd609ccb06f98ef955fed561c",
        "domain": "numina",
        "question": "80. In how many ways can three identical items be distributed among ten people if there is no limit to the number of items offered to one person?",
        "answer": "220",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44901,
        "difficulty": null
      },
      {
        "id": "06e92be7720a6517e0324d265337b878ecf75495e414d3b1283971c799251d2f",
        "domain": "numina",
        "question": "5. Find the smallest positive integer $n$ such that: for any $n$ points $A_{1}, A_{2}, \\cdots, A_{n}$ taken on the circumference of circle $O$, among the $\\mathrm{C}_{n}^{2}$ angles $\\angle A_{i} O A_{j}(1 \\leqslant i<j \\leqslant n)$, at least 2007 angles are not greater than $120^{\\circ}$.",
        "answer": "91",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146044,
        "difficulty": null
      },
      {
        "id": "f53408ea5242c491f2bf931abd9ce4e3202f361915901d6ed3ed12807d87d849",
        "domain": "numina",
        "question": "B4. The candy store sells chocolates in the flavors white, milk, and dark. You can buy them in three different colored boxes. The three colored boxes have the following contents:\n\n- Gold: 2 white, 3 milk, 1 dark,\n- Silver: 1 white, 2 milk, 4 dark,\n- Bronze: 5 white, 1 milk, 2 dark.\n\nLavinia buys a number of chocolate boxes (at least one) and upon returning home, it turns out she has exactly the same number of chocolates of each flavor.\n\nWhat is the minimum number of boxes Lavinia could have bought?",
        "answer": "20",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85504,
        "difficulty": null
      },
      {
        "id": "9c8edf5549bbe1633b613774861aeb05cf9a45b8428d2eb2fff91cb97fec2b5f",
        "domain": "numina",
        "question": "## Task 4.\n\nEach field of a $9 \\times 9$ board is colored either blue or red. A row is said to be predominantly red if it has more red than blue fields, and a column is said to be predominantly blue if it has more blue than red fields. Let $C$ be the number of predominantly red rows, and $P$ be the number of predominantly blue columns.\n\na) Can it be that $C+P=18$?\n\nb) Can it be that $C+P=17$?\n\nc) Can it be that $C+P=16$?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 90074,
        "difficulty": null
      },
      {
        "id": "229ec13a7f651b427d2f4d00143a027f54b7afc1876e00fa98b23d7a271b38a5",
        "domain": "numina",
        "question": "# Task № 3.4\n\n## Condition:\n\nCupcakes are sold in boxes of 2, eclairs - in boxes of 6, and gingerbread - in boxes of 15. You can only buy whole boxes, you cannot open them. Alice bought an equal number of cupcakes, eclairs, and gingerbread. What is the smallest number of boxes she could have taken",
        "answer": "22",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13887,
        "difficulty": null
      }
    ],
    "anchor_candidates": [
      {
        "id": "e911f3e372161917410d5b0ffac759b5a9a499cfef36ef5863513ca82a5cf87e",
        "domain": "numina",
        "question": "14. Let the complex numbers $z_{1}, z_{2}$ satisfy\n$$\n\\begin{array}{l}\n\\left|z_{1}\\right|=\\left|z_{2}\\right|=1 \\text {, and } z_{1}+z_{2}-1=0 \\text {. } \\\\\n\\text { Then }\\left(z_{1}-z_{2}\\right)^{2}=\n\\end{array}\n$$",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92306,
        "difficulty": null
      },
      {
        "id": "f2d04826f7f4d15256c1c3e7c7a41f267c510a6bd976f018d0fa1e4672b0f3ce",
        "domain": "numina",
        "question": "## 53. Math Puzzle 10/69\n\nAt the beginning of a group meeting, the Pioneers greet each other with handshakes. A total of 66 handshakes are exchanged.\n\nHow many friends attended the meeting?",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 181745,
        "difficulty": null
      },
      {
        "id": "62355777ca148c4d97cd349c2518dafdbc0715df0432f8515dfda93bdc00b6f7",
        "domain": "numina",
        "question": "A1. At a bazaar, you can win a prize by guessing the exact number of ping pong balls in a glass jar. Arie guesses there are 90, Bea guesses there are 97, Cor guesses there are 99, and Dirk guesses there are 101. None of the four win the prize. It turns out that one of the four is off by 7, one is off by 4, and one is off by 3.\n\nHow many ping pong balls are in the jar?",
        "answer": "94",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 85839,
        "difficulty": null
      },
      {
        "id": "fd1ff5c6ac4a40617013cb6b07286bca6453fd346b2b873a548528d20428ab64",
        "domain": "numina",
        "question": "## Task A-1.6.\n\nGiven is a triangle $A B C$ with an area of 1. Point $D$ is the midpoint of segment $\\overline{B C}$, point $E$ is the midpoint of segment $\\overline{A D}$, point $F$ is the midpoint of segment $\\overline{B E}$, and point $G$ is the midpoint of segment $\\overline{C F}$. Determine the area of triangle $E F G$.",
        "answer": "\\frac{1}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87636,
        "difficulty": null
      },
      {
        "id": "9c863d1a3dc76a31671983041d57bb4b3331c52c5318b02e4a69e8fea534a069",
        "domain": "numina",
        "question": "18. Find the remainder when $2^{55}+1$ is divided by 33 .",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160273,
        "difficulty": null
      },
      {
        "id": "d03b5399aadf9b543ec8f1036821e73be93f24f6a61f127977cc2fc7e19312cd",
        "domain": "numina",
        "question": "4. (2004 National College Entrance Examination - National Paper) The maximum value of the function $y=\\sin x-\\frac{1}{2} \\cos x(x \\in \\mathbf{R})$ is",
        "answer": "\\frac{\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116412,
        "difficulty": null
      },
      {
        "id": "b755391cf244ac4e0ca83fd1d2990360f23e1cd07eb83f5f5e3ff4bd48896a76",
        "domain": "numina",
        "question": "72. Find the product:\n\n$$\n\\left(1-\\frac{1}{4}\\right) \\cdot\\left(1-\\frac{1}{9}\\right) \\cdot\\left(1-\\frac{1}{16}\\right) \\ldots\\left(1-\\frac{1}{225}\\right)\n$$",
        "answer": "\\frac{8}{15}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39562,
        "difficulty": null
      },
      {
        "id": "5a74e52d8195264538bba7b54cacffa279357226c1893de6141b4fcf4d7a02bb",
        "domain": "numina",
        "question": "5. For the distinct real numbers $a, b, c$, it holds that\n\n$$\n\\frac{a^{2}}{(b-c)^{2}}+\\frac{b^{2}}{(c-a)^{2}}+\\frac{c^{2}}{(a-b)^{2}}=2\n$$\n\nCalculate\n\n$$\n\\frac{a}{b-c}+\\frac{b}{c-a}+\\frac{c}{a-b}\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189329,
        "difficulty": null
      },
      {
        "id": "0c6621092f15b2bab7ac62ae6bd15038e98e77b1768a1591cdccb93fa43746c7",
        "domain": "numina",
        "question": "3. The residents of the Immortal Island celebrate their birthdays every 3 years. Xiaohuaxian had her first birthday at the age of 1, and her second birthday at the age of 4. She receives a number of gifts equal to her age each birthday. By the time she finished celebrating her birthday this year, Xiaohuaxian had received a total of 70 birthday gifts. How old is Xiaohuaxian this year? $\\qquad$ years old.",
        "answer": "19",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 120911,
        "difficulty": null
      },
      {
        "id": "647a7ce29431e3fbf05408858ed2f98ba2eb336d7f9d1eaed8d51324f1e0a58d",
        "domain": "numina",
        "question": "8. Given that the three sides of $\\triangle A B C$ are $13,14,15$. There are 4 circles $O, O_{1}, O_{2}, O_{3}$ with the same radius $r$ placed inside $\\triangle A B C$, and circle $O_{1}$ is tangent to sides $A B$ and $A C$, circle $O_{2}$ is tangent to sides $B A$ and $B C$, circle $O_{3}$ is tangent to sides $C B$ and $C A$, and circle $O$ is tangent to circles $O_{1}, O_{2}, O_{3}$. Then $r=$ $\\qquad$ .",
        "answer": "\\frac{260}{129}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98050,
        "difficulty": null
      },
      {
        "id": "cdb9b8804f9782f4d569770722ad5a48feb0a7307da413c59adfd735734aa81c",
        "domain": "numina",
        "question": "【Question 10】\nThe school is to distribute 90 storybooks to third-grade students. If distributed at a rate of 1 book per student, there will be leftovers; if the remaining books are then distributed at a rate of 1 book for every two students, they will be exactly distributed. Therefore, the number of third-grade students is $\\qquad$ people.",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 121671,
        "difficulty": null
      },
      {
        "id": "f005c1fb601e49073c8ff35b49a16f07e8580eb676dbcf7f139890ab354b32ef",
        "domain": "numina",
        "question": "2. For $x, y \\in[1,3]$ find the minimum value of the expression\n\n$$\nA=\\frac{\\left(3 x y+x^{2}\\right) \\sqrt{3 x y+x-3 y}+\\left(3 x y+y^{2}\\right) \\sqrt{3 x y+y-3 x}}{x^{2} y+y^{2} x}\n$$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8016,
        "difficulty": null
      },
      {
        "id": "e647f058cd215c3e3d65651c9cd8a6f5dd0ac7926ae4444e0a27ce86f73745ca",
        "domain": "numina",
        "question": "(7) Set $A=\\left\\{x \\left\\lvert\\, x=\\left[\\frac{5 k}{6}\\right]\\right., k \\in \\mathbf{Z}, 100 \\leqslant k \\leqslant 999\\right\\}$, where $[x]$ denotes the greatest integer not greater than $x$. Then the number of elements in set $A$ is $\\qquad$ .",
        "answer": "750",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 98302,
        "difficulty": null
      },
      {
        "id": "a59b238bc1c00f47702e2e393e25d17ed21984a8f63f78e3d55ec66f14c668bc",
        "domain": "numina",
        "question": "1. Find all prime numbers $p$ for which the number $7^{p-2}+9 p^{4}$ is a perfect square.",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162738,
        "difficulty": null
      },
      {
        "id": "0754186b45247ea9a80b843828d8d0550782dee24776fecc1d15d5a8a9d9f44d",
        "domain": "numina",
        "question": "3. In $\\triangle A B C$, the lengths of the sides opposite to $\\angle A$, $\\angle B$, and $\\angle C$ are $a$, $b$, and $c$ respectively. Point $G$ satisfies\n$$\n\\overrightarrow{G A}+\\overrightarrow{G B}+\\overrightarrow{G C}=\\mathbf{0}, \\overrightarrow{G A} \\cdot \\overrightarrow{G B}=0 \\text {. }\n$$\n\nIf $(\\tan A+\\tan B) \\tan C=m \\tan A \\cdot \\tan B$, then $m=$ . $\\qquad$",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 118759,
        "difficulty": null
      },
      {
        "id": "e12fbe44263f2dbbf0988e650c2761f32ad1812ee3fda02e9a2230565c020502",
        "domain": "numina",
        "question": "In the picture, there is a structure composed of twelve identical cubes. To how many different places can we move the dark cube so that the surface area of the assembled body does not change?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_22cb57ce78fb8d5dd5c3g-5.jpg?height=334&width=394&top_left_y=1758&top_left_x=837)\n\nJust like in the original structure, the cubes in the new structure must touch each other with entire faces. The positions of the light cubes cannot be changed.\n\n(D. Reichmann)\n\nHint. How will the surface area of the body change after removing the dark cube?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76219,
        "difficulty": null
      },
      {
        "id": "d09fe9b368eb88461e14d595ec32e16c5e3ecb28554ac8b37f5bc3aab8fb8ada",
        "domain": "numina",
        "question": "Let's divide a solid cube with an edge of $n \\mathrm{~cm}$ into smaller cubes with an edge of $1 \\mathrm{~cm}$ ( $n$ is a natural number). For which $n$ can a cube with an edge of $2 n \\mathrm{~cm}$ be assembled from the resulting small cubes, if we allow there to be a void inside the cube?",
        "answer": "23",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171750,
        "difficulty": null
      },
      {
        "id": "2bd069f2491cc0df59149e7fd477d143e67da92e364901055ba31538911d2500",
        "domain": "numina",
        "question": "6. Find the sum of the integers that belong to the set of values of the function $f(x)=\\log _{3}(40 \\cos 2 x+41)$ for $x \\in[(5 / 3)(\\operatorname{arctg}(1 / 5)) \\cos (\\pi-\\arcsin (-0.8)) ; \\operatorname{arctg} 3]$ (10 points)",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10495,
        "difficulty": null
      },
      {
        "id": "ec7fe7cbb5abcf0185fef22e681f4cbc4a8ef3081d94edd5366107cc3ced63f1",
        "domain": "numina",
        "question": "5. Option 1.\n\nPetya is the oldest child in the family. He has two sisters, Anya and Katya, and a brother Vasya. Petya calculated that Anya and Katya together are 19 years old, and Anya and Vasya together are 14 years old. Determine how old Katya is, if it is known that the two youngest children together are 7 years old.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13133,
        "difficulty": null
      },
      {
        "id": "ce12c1ece3e21019a5d0dd430b001ba45d974bfcac9753ae76b84b11ed8d0718",
        "domain": "numina",
        "question": "## Task 3 - 190833\n\nGiven a square $A B C D$ with side length $a$.\n\nA line parallel to $A B$ intersects the sides $B C$ and $A D$ at points $E$ and $F$, respectively. A line parallel to $B C$ intersects $A B$ and $E F$ at points $G$ and $H$, respectively, and a line parallel to $A B$ intersects the segments $B E$ and $G H$ at points $J$ and $K$, respectively.\n\na) Determine the perimeter of the rectangle $K J E H$ in terms of $a$ under the condition that the rectangles $A G H F, G B J K, K J E H$, and $F E C D$ are equal in area!\n\nb) Determine the area of the rectangle $K J E H$ in terms of $a$ under the condition that the rectangles $A G H F, G B J K, K J E H$, and $F E C D$ are equal in perimeter!",
        "answer": "\\frac{3}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 184151,
        "difficulty": null
      },
      {
        "id": "974593d6330fd9c4735568aafd853aebcff7b3e48d805dc7d1f23678b8883d53",
        "domain": "numina",
        "question": "A straight prism has a rhombus as its base, with an acute angle of $60^{\\circ}$. What angle does a plane form with the base if the intersection of this plane with the prism's lateral surface is a square?",
        "answer": "54.7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171091,
        "difficulty": null
      },
      {
        "id": "31be999f0612e1a2e97b61b940b7b2f73cdff51e9684b71bea9162852d245ac0",
        "domain": "numina",
        "question": "4. In the tetrahedron $P-ABC$, it is known that $BC=3$, $CA=4$, $AB=5$. If the dihedral angles between the three lateral faces and the base are all $45^{\\circ}$, then the volume of the tetrahedron $P-ABC$ is $\\qquad$",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 111512,
        "difficulty": null
      },
      {
        "id": "0d3f9f1d9689828f472fd027ff703ee348c870686d88dd2f76497407b0f99b1d",
        "domain": "numina",
        "question": "Example 5 Let the function $f(x)$ be defined for all $x>0$, and satisfy the following conditions:\n(1) For $x>0$, $f(x) f\\left[f(x)+\\frac{1}{x}\\right]=1$; (2) $f(x)$ is increasing on $(0,+\\infty)$. Find $f(1)$.",
        "answer": "\\frac{1-\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100972,
        "difficulty": null
      },
      {
        "id": "189abd25fdbd31a74f5b492f7e7f1fc4ab767602d1fa6335cace14959706f8f8",
        "domain": "numina",
        "question": "1. Given the quadratic function\n$$\nf(x)=x^{2}+b x+8(b \\neq 0)\n$$\n\nhas two distinct real roots $x_{1}, x_{2}$, and $x_{1}+\\frac{1}{x_{2}}, x_{2}+\\frac{1}{x_{1}}$ are the two zeros of a quadratic function $g(x)$ with a leading coefficient of 1. If $g(1)=f(1)$, find all possible values of $g(1)$.",
        "answer": "-8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138175,
        "difficulty": null
      },
      {
        "id": "bdf65986d82d3d74d766253da1c41ba598b671a516ac38588723f899fd081f1f",
        "domain": "numina",
        "question": "## Task 4\n\nIt is known about a three-digit number: It is less than 200 and ends with the digit 5. The middle digit is the smallest natural number. What is the number?",
        "answer": "105",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 186699,
        "difficulty": null
      },
      {
        "id": "a4d27d749bb21a95dede5019c364003f9ce7c67a3f3f4c1b2c8faa825b4ba363",
        "domain": "numina",
        "question": "9. Find the smallest natural number that is greater than the sum of its digits by 1755 (the year of the founding of Moscow University).",
        "answer": "1770",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6318,
        "difficulty": null
      },
      {
        "id": "cdf0299cea1aea7ad034827922c24b9bc1e43de009c228a5af76ccbef3235f1c",
        "domain": "numina",
        "question": "3.18. Calculate\n\n$$\n\\int_{L}\\left(z^{2}+2 z \\bar{z}\\right) d z\n$$\n\nwhere $L-$ is the arc of the circle $|z|=1, \\arg z \\in[0, \\pi]$.",
        "answer": "-\\frac{14}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30767,
        "difficulty": null
      },
      {
        "id": "3cf7a12e3e26cf73233c1620f4f5d32401a3de406d0fdf8a3c209e022776bd13",
        "domain": "numina",
        "question": "18. A collection of 2011 circles divide the plane into $N$ regions in such a way that any pair of circles intersects at two points and no point lies on three circles. Find the last four digits of $N$.",
        "answer": "2112",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160753,
        "difficulty": null
      },
      {
        "id": "11ad4b4e48db0c56d16a2e27c2006aa60e43e94bfcf9307c62e930a9992a4881",
        "domain": "numina",
        "question": "## Task 2\n\nCalculate the difference between the product and the sum of the numbers 9 and 2.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186446,
        "difficulty": null
      },
      {
        "id": "b71de2d4611def905ce21efe6dec54a61389876cf42c8a9d4a917c1c937f2daf",
        "domain": "numina",
        "question": "4. For each pair of numbers $x$, let us denote by $s(x, y)$ the smallest of the numbers $x, 1-y, y-x$. What is the largest value that the number $s(x, y)$ can take?",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6756,
        "difficulty": null
      },
      {
        "id": "ef0a841e698cac73389dfb73cafe5cb732b43508c04ec71139e2b94f9db412dc",
        "domain": "numina",
        "question": "1. Find the maximum possible area of a quadrilateral in which the product of any two adjacent sides is 1.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12059,
        "difficulty": null
      },
      {
        "id": "7f7b925eaff45fb89b7bea67ea2280479c7979dcf69cab20c11507f75a8d6d05",
        "domain": "numina",
        "question": "$3+$ [ Law of Sines\n\nIn an isosceles trapezoid, the bases are given as $a=21, b=9$ and the height $h=8$. Find the radius of the circumscribed circle.\n\n#",
        "answer": "\\frac{85}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63232,
        "difficulty": null
      },
      {
        "id": "de54ae0eee46d4816418fedf6846633019ac1617f57fd030cd442427ca884899",
        "domain": "numina",
        "question": "How many obtuse triangles are there in which the measures of the angles are three different integers?",
        "answer": "1936",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174515,
        "difficulty": null
      },
      {
        "id": "757950eb5e3f0ea5504a2598e634957737ef7dfca5542216f776bfa76f7a8e60",
        "domain": "numina",
        "question": "7. Given a complex number $z$ satisfying $|z|=1$. Then the maximum value of $\\left|z^{3}-3 z-2\\right|$ is $\\qquad$ .\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "3\\sqrt{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 101701,
        "difficulty": null
      },
      {
        "id": "3f6287b4d8873c8b7c93aa848c1d83e6bdc00a95b8d4b3ceb6db470d968b589e",
        "domain": "numina",
        "question": "3. (10 points) The simplest fraction $\\frac{\\mathrm{a}}{\\mathrm{b}}$ satisfies $\\frac{1}{5}<\\frac{\\mathrm{a}}{\\mathrm{b}}<\\frac{1}{4}$, and $b$ does not exceed 19, then the product of the maximum possible value and the minimum possible value of $a+b$ is",
        "answer": "253",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100633,
        "difficulty": null
      },
      {
        "id": "5c09b17f5979a23d5c018861820531b4c2872afbfc7402db959141fe1e88dfa2",
        "domain": "numina",
        "question": "11. In the Cartesian coordinate system $x O y$, the circle $\\odot M$ with center $M(0,1)$ intersects the parabola $y=x^{2}$ at four points $A$, $B$, $C$, and $D$ in sequence.\n(1) Find the range of the radius $r$ of $\\odot M$;\n(2) Find the maximum area of quadrilateral $A B C D$ (accurate to $10^{-4}$).",
        "answer": "1.0887",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105699,
        "difficulty": null
      },
      {
        "id": "c60d88ef0a62d5056d033d85fb22d76c772f5f0c2cb62d2025e2eefbea0e647b",
        "domain": "numina",
        "question": "34. [20] Let $f(n)$ be the largest prime factor of $n$. Estimate\n$$\nN=\\left\\lfloor 10^{4} \\cdot \\frac{\\sum_{n=2}^{10^{6}} f\\left(n^{2}-1\\right)}{\\sum_{n=2}^{10^{6}} f(n)}\\right\\rfloor .\n$$\n\nAn estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{3}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
        "answer": "18215",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151889,
        "difficulty": null
      },
      {
        "id": "a12635bbd434a8d8a267f466b4fbf647229632588659c132b9bff51f47e9ad54",
        "domain": "numina",
        "question": "1.1. (2 points) On two bookshelves, there are mathematics books, with an equal number on each. If 5 books are moved from the first shelf to the second, then the second shelf will have twice as many books as the first. How many books are there in total on both shelves?",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6440,
        "difficulty": null
      },
      {
        "id": "1e01658a0988ef648e3aa99f6f98ff9eac4796284a4b1785c1f8c30ea5096e9e",
        "domain": "numina",
        "question": "5. Let $f(x)=a_{0}+a_{1} x+a_{2} x^{2}+a_{4} x^{3}+a_{4} x^{4}+a_{5} x^{5}$ be a polynomial in $x$ where $a_{0}, a_{1}, a_{2}, a_{3}, a_{4}$ are constants and $a_{5}=7$. When divided by $x-2004, x-2005, x-2006, x-2007$ and $x-2008$, $\\mathrm{f}(x)$ leaves a remainder of $72,-30,32,-24$ and 24 respectively. Find the value of $f(2009)$.",
        "answer": "1742",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160521,
        "difficulty": null
      },
      {
        "id": "af2ae931cd4f4bc27a7217f25f62068fdc025eb98b05f6d0f8b35f18657ca2b2",
        "domain": "numina",
        "question": "Example 9. The probability of an event occurring in each of 900 independent trials is 0.5. Find the probability that the relative frequency of the event's occurrence deviates from its probability in absolute value by no more than 0.02.",
        "answer": "0.7698",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 33135,
        "difficulty": null
      },
      {
        "id": "d3e008b2f11d995086755552eb65d8f9dae277165730b3afad0c8a768a34bf48",
        "domain": "numina",
        "question": "## Task 2\n\nIn a district, there were 1685 apartments. In the last five years, 275 apartments were built there. How many apartments are there now in this district?",
        "answer": "1960",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186636,
        "difficulty": null
      },
      {
        "id": "5941de0092286d31199e0bfd35fe8ea2e6e39fe939aa8dee6541f838564dfe46",
        "domain": "numina",
        "question": "As shown in the figure, in quadrilateral $A B C D$, $\\angle B=90^{\\circ}, \\angle D A B=\\angle D=96^{\\circ}$, $\\angle B C D=78^{\\circ}$, and $D A=2 A B$, find $\\angle C A B$.",
        "answer": "66",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122407,
        "difficulty": null
      },
      {
        "id": "56eafe826df28b512ff55cec2d306d68313f9d53aee36a80d4125582e82d4e2e",
        "domain": "numina",
        "question": "14. (6 points) A group of PLA soldiers formed a three-layer hollow square formation with 9 soldiers left over. If another layer was added inside the hollow part, 7 soldiers would be short. The total number of soldiers in this group is $\\qquad$ people.",
        "answer": "105",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 102854,
        "difficulty": null
      },
      {
        "id": "cff9f7e4c369ba62df3434f6648fbb99e6d1a134067130a779de31c6cb011e11",
        "domain": "numina",
        "question": "7. Let $m$ and $n$ both be positive integers greater than 1. Given that $\\frac{(n+m)!}{n!}=17297280$. Then $\\frac{n}{m}=$ $\\qquad$",
        "answer": "\\frac{31}{2}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 103131,
        "difficulty": null
      },
      {
        "id": "739b26f05fa881642b3b8ac28e114305941c24a76446ceb6609c29557dbcf367",
        "domain": "numina",
        "question": "16. (5 points)\nThe Reuleaux triangle is a famous shape. By using the vertices of an equilateral triangle as the centers and its side length as the radius to draw three circular arcs, a Reuleaux triangle can be formed. In the figure below, the perimeter of the Reuleaux triangle is $314 \\mathrm{~cm}$, then the side length of the central equilateral triangle is $\\qquad$ $\\mathrm{cm}$. ( $\\pi$ is taken as 3.14)",
        "answer": "100",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 120924,
        "difficulty": null
      },
      {
        "id": "162e06ffe4003940e20b8402e99c6116bf796a818465c8c9679d0b8a5a243151",
        "domain": "numina",
        "question": "Problem 9.5. On the base $AC$ of isosceles triangle $ABC (AB = BC)$, a point $M$ is marked. It is known that $AM = 7, MB = 3, \\angle BMC = 60^\\circ$. Find the length of segment $AC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-33.jpg?height=240&width=711&top_left_y=86&top_left_x=369)",
        "answer": "17",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13448,
        "difficulty": null
      },
      {
        "id": "f4328185556075549c0b8f9bac98c4fe61b6371f76dde210cd40157affaf732b",
        "domain": "numina",
        "question": "I1.1 The given figure shows a circle of radius $18 \\mathrm{~cm}$, centre $O$.\nIf $\\angle A O B=\\frac{\\pi}{3}$ and the length of arc $A P B$ is $a \\pi \\mathrm{cm}$, find $a$.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152557,
        "difficulty": null
      },
      {
        "id": "8d525d938408bb1a0288939f6d0a64cf67be04cf51dff1875bf5b923b247c417",
        "domain": "numina",
        "question": "12.55*. In a right triangle $ABC$ with a right angle at $A$, a circle is constructed on the altitude $AD$ as a diameter, intersecting side $AB$ at point $K$ and side $AC$ at point $M$. Segments $AD$ and $KM$ intersect at point $L$. Find the acute angles of triangle $ABC$, given that $AK: AL = AL: AM$.",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 27594,
        "difficulty": null
      },
      {
        "id": "420dcaf0c1d328c64318b67bb621d97e6047369b1007bbb4ae020cd8a112bfe0",
        "domain": "numina",
        "question": "$\\begin{array}{l}\\text { 1. Let } p, q \\in \\mathbf{R}_{+} \\text {, and satisfy } \\\\ \\log _{9} p=\\log _{12} q=\\log _{16}(p+q) \\text {. } \\\\ \\text { Then } \\frac{q}{p}=\\end{array}$",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142207,
        "difficulty": null
      },
      {
        "id": "206dc188d17a35c9a05a6d007b64e2104b8383ca266ae2ad9eec7e246e5fe120",
        "domain": "numina",
        "question": "$10 、$ Use the four numbers $8 、 8 、 8 、 10$, and the four basic arithmetic operations to form an expression $\\qquad$ that equals 24 (parentheses can be added)",
        "answer": "24",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 101852,
        "difficulty": null
      },
      {
        "id": "d3791abdef2470bc6ff4e7a5aa5d93c7815890d10f61409427649bb0f775b100",
        "domain": "numina",
        "question": "12. Given the line $a x+b y+c=0$ where $a, b, c$ are 3 different elements taken from the set $\\{-3,-2$, $-1,0,1,2,3\\}$, and the inclination angle of the line is acute. Then the number of such lines is $\\qquad$",
        "answer": "43",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 109452,
        "difficulty": null
      },
      {
        "id": "477e87a861cdb0201856813c416a54677672ead44521eaa10708b9aa8f074150",
        "domain": "numina",
        "question": "## Task A-4.2.\n\nLet $S$ be the set of all natural numbers less than 1000 whose all digits in decimal notation are even. Let $\\omega$ be a complex number such that $\\omega^{2}+\\omega+1=0$.\n\nCalculate the sum $\\sum_{k \\in S} \\omega^{k}$, i.e., the sum of the values $\\omega^{k}$ for all $k$ in the set $S$.",
        "answer": "-2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88871,
        "difficulty": null
      },
      {
        "id": "9bb710d2190a6b3bab02b5beec2c4e3dd46606647f45c7e366587da90db5eebd",
        "domain": "numina",
        "question": "A mother distributes walnuts from a basket among her 4 children. Tamás receives one walnut and a quarter of the remaining walnuts; then Erzsi receives one walnut and a quarter of the remaining walnuts; Béla also receives one walnut and a quarter of the remaining walnuts; finally, Juliska receives one walnut and a quarter of the remaining walnuts. It turned out that the boys received 100 more walnuts than the girls. How many walnuts were there originally, and how many were left in the basket at the end?",
        "answer": "1021",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 176589,
        "difficulty": null
      },
      {
        "id": "c6476a0fec1df270cea356b4fd4a71cf1bef341a8607416dd4deb43bab2c256f",
        "domain": "numina",
        "question": "Find the smallest natural number such that its half is divisible by three, its third is divisible by four, its quarter is divisible by eleven, and its half gives a remainder of five when divided by seven.\n\n(E. Patáková)",
        "answer": "528",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76759,
        "difficulty": null
      },
      {
        "id": "6fee52661dff1bbd667e5a302a6c68c5adfe7b292c578c81e329381b9ad96e28",
        "domain": "numina",
        "question": "[Algebraic equations and systems of equations (other)]\nFormulas for abbreviated multiplication (other)\n\nThe number $a$ is a root of the equation $x^{11} + x^{7} + x^{3} = 1$. For which natural values of $n$ does the equality $a^{4} + a^{3} = a^{n} + 1$ hold?",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69646,
        "difficulty": null
      },
      {
        "id": "16cddfe5b14baf5ffe3c3b56b08daf0d745d5a7d3db401529282485fa30e7bd9",
        "domain": "numina",
        "question": "3. Solve the equation $2021 x=2022 \\cdot \\sqrt[2022]{x^{2021}}-1 .(10$ points $)$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19070,
        "difficulty": null
      },
      {
        "id": "e531efbf86d48bf18a51e192fdbff5753f9601da4b083c798773b567fb08352a",
        "domain": "numina",
        "question": "3.5. Znayka cut out a semicircle from paper. Neznaika marked a point $C$ on the diameter $A B$ of this semicircle and cut off two semicircles from Znayka's semicircle with diameters $A C$ and $C B$. The area of the remaining figure turned out to be $10 \\pi$. Find the length of the part of the chord lying inside it, passing through point $C$ perpendicular to $A B$. If necessary, round the answer to two decimal places.",
        "answer": "2\\sqrt{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5819,
        "difficulty": null
      },
      {
        "id": "8341dad95c2e4b14edf77884662230612f8fa88089bda28ba8ffb70ce02b4f69",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{\\pi} 2^{4} \\cdot \\cos ^{8} x d x\n$$",
        "answer": "\\frac{35\\pi}{8}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46292,
        "difficulty": null
      },
      {
        "id": "e6669dca6538b15f15b219df1eceba86a9693c3fa135758bab2f959f04c88f79",
        "domain": "numina",
        "question": "8. (12 points) Fill the ten digits from $0-9$ into the circles in the figure so that the sums of the numbers on the five lines form an arithmetic sequence, and the sum of the terms of this arithmetic sequence is 55. The number of possible values for the common difference of this arithmetic sequence is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "3",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 134862,
        "difficulty": null
      },
      {
        "id": "62fcb72ea1ed2fd68d7cb5f21f31e2f9fe087adca620894209912e2d687f2699",
        "domain": "numina",
        "question": "1. (6 points) Calculate: $\\frac{2 \\frac{1}{4}+0.25}{2 \\frac{3}{4}-\\frac{1}{2}}+\\frac{2 \\times 0.5}{2 \\frac{1}{5}-\\frac{2}{5}}=$",
        "answer": "\\frac{5}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139307,
        "difficulty": null
      },
      {
        "id": "05a1b61de3e4f12a0702c075640d0f3392cc1dbba031a979106d239b0633fad3",
        "domain": "numina",
        "question": "16. (6 points) A road is 400 meters long, and on both sides of the road, a trash can is placed every 20 meters, with the start and end points being bus stops where no trash cans are placed. How many trash cans are placed in total?",
        "answer": "38",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 106700,
        "difficulty": null
      },
      {
        "id": "3ee54b88d77d27c98d690a5610378d57ea01676c3085077212670df50c8b2813",
        "domain": "numina",
        "question": "14. Let $A=\\{1,2, \\ldots, 2011\\}$. Find the number of functions $f$ from $A$ to $A$ that satisfy $f(n) \\leq n$ for all $n$ in $A$ and attain exactly 2010 distinct values.",
        "answer": "2^{2011}-2012",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151320,
        "difficulty": null
      },
      {
        "id": "d29a08cc29581657d094ff4d714454f0990b097815a2d394bca700b5d90f7659",
        "domain": "numina",
        "question": "3. (7 points) Four mole burrows $A, B, C, D$ are connected sequentially by three tunnels. Each minute, the mole runs through a tunnel to one of the adjacent burrows. In how many ways can the mole get from burrow $A$ to $C$ in 20 minutes?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d5d5613c0f147d056827g-22.jpg?height=103&width=834&top_left_y=288&top_left_x=608)",
        "answer": "6765",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19684,
        "difficulty": null
      },
      {
        "id": "ad3b4b454d0f04b2cbe4eeeba15933034519d2f6371d609029b02dad2b0ba338",
        "domain": "numina",
        "question": "The geometric progressions $a_{1}, a_{2}, a_{3}, \\ldots$ and $b_{1}, b_{2}, b_{3}, \\ldots$ have the same ratio, with $a_{1}=$ $27, b_{1}=99$ and $a_{15}=b_{11}$. Find the value of $a_{9}$.\n\n#",
        "answer": "363",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73455,
        "difficulty": null
      },
      {
        "id": "468f64f8c8b9aa51659c3d916f1834355b74ba6e0e61ac8bc0258002a147c2e2",
        "domain": "numina",
        "question": "The reverse of a two-digit integer is the number obtained by reversing the order of its digits. For example, 34 is the reverse of 43. How many numbers exist such that when added to their reverse, they give a perfect square?",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 73256,
        "difficulty": null
      },
      {
        "id": "2bd4593d35930746417e802b0b1d80248981945cd563812313fa52d035f01df3",
        "domain": "numina",
        "question": "The equation  \n$$ (x-1)(x-2) \\cdots(x-2016)=(x-1)(x-2) \\cdots(x-2016) $$  \nis written on the board. One tries to erase some linear factors from both sides so that each side still has at least one factor, and the resulting equation has no real roots. Find the least number of linear factors one needs to erase to achieve this.",
        "answer": "2016",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 196270,
        "difficulty": null
      },
      {
        "id": "d3d2cc4ea09616d5185fb94a541cd878d44d9ae16366273cc665b5c3869325a6",
        "domain": "numina",
        "question": "3. (3 points) There are two cylinders with a volume ratio of 5:8, and the lateral surfaces of these cylinders unfold into the same rectangle. If the length and width of this rectangle are both increased by 6, its area increases by 114. What is the area of this rectangle $\\qquad$ .",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 121481,
        "difficulty": null
      },
      {
        "id": "971c848b35c4c32ebf4f1207ddb68ff806cfa4e6005e15657dee7fa9ef4a813a",
        "domain": "numina",
        "question": "On a circle of radius 1, a point $O$ is marked, and from it, a notch is made to the right with a radius of $l$. From the resulting point $O_{1}$, another notch is made in the same direction with the same radius, and this is repeated 1968 times. After this, the circle is cut at all 1968 notches, resulting in 1968 arcs. How many different lengths of arcs can be obtained in this way?",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 53876,
        "difficulty": null
      },
      {
        "id": "cd9f86ea9b950d0d4f52e9c0c489df15ecc253c2ad1997efdbab5df7f4a103a4",
        "domain": "numina",
        "question": "5. Given is $\\triangle A B C, \\measuredangle C=90^{\\circ}$. Squares $A E D C$ and $C F G B$ are constructed externally on its sides. Segment $E B$ intersects segments $A C$ and $A G$ at points $H$ and $I$, respectively. Segment $A G$ intersects segment $B C$ at point $J$. If the area of $\\triangle A I B$ is 2017, find the area of quadrilateral $H C J I$.\n\n## SOLUTIONS TO THE MUNICIPAL COMPETITION IN MATHEMATICS FOR HIGH SCHOOL STUDENTS, 21. 1. 2017.\n\n## First Grade - A category",
        "answer": "2017",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 161509,
        "difficulty": null
      },
      {
        "id": "e248e03b55fddf8fa542ff76db6bc3a954655c9f0254cb6e86fa7be5619c071f",
        "domain": "numina",
        "question": "Let $x$ be a non-zero real number such that $x+\\frac{1}{x}=\\sqrt{2019}$. What is the value of $x^{2}+\\frac{1}{x^{2}}$?",
        "answer": "2017",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 79273,
        "difficulty": null
      },
      {
        "id": "453e4c4848c2daee19dedcfdae77102e013946b20659076b81112f91223dabc6",
        "domain": "numina",
        "question": "In the plane, there are 2020 points, some of which are black and the rest are green.\n\nFor each black point, there are exactly two green points that are at a distance of 2020 from this black point.\n\nDetermine the smallest possible number of green points.\n\n(Walther Janous)\n\nAnswer. The smallest possible number is 45 green points.",
        "answer": "45",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 190959,
        "difficulty": null
      },
      {
        "id": "8065a1d7a8280aa9bc5ed19e84e63d7a76562b0702c61cb325e64b6fe0d791b5",
        "domain": "numina",
        "question": "71. A three-digit number leaves a remainder of 2 when divided by 4, 5, and 6. If three digits are added to the end of this number to make it a six-digit number, and this six-digit number is divisible by $4, 5, 6$, then the smallest six-digit number that meets the condition is",
        "answer": "122040",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 132832,
        "difficulty": null
      },
      {
        "id": "9e45c0bfef3cda6bac3d0c7a29f55fdcacfbd20ce91a48c006b6e623398002ad",
        "domain": "numina",
        "question": "$4.35 \\operatorname{tg} 435^{\\circ}+\\operatorname{tg} 375^{\\circ}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.\n\n$4.35 \\tan 435^{\\circ}+\\tan 375^{\\circ}$.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49628,
        "difficulty": null
      },
      {
        "id": "ef9b57b17c164a0f8604a415113c834c3688246fed114676db827eeea2875686",
        "domain": "numina",
        "question": "# Problem 3. (3 points)\n\nAll residents of the island are either blondes or brunettes with green or blue eyes. The proportion of brunettes among blue-eyed residents is $65 \\%$. The proportion of blue-eyed residents among blondes is $70 \\%$. Finally, the proportion of blondes among green-eyed residents is 10\\%. What percentage of the island's population are green-eyed brunettes?",
        "answer": "54",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 20651,
        "difficulty": null
      },
      {
        "id": "102bc0bec38ce3bb1c073cdff738170f9801ed3f42943fa8ffea3ab302a64f38",
        "domain": "numina",
        "question": "99. In a regular $n$-gon ( $n>5$ ), the difference between the largest and smallest diagonals is equal to the side. Find $n$.",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 25304,
        "difficulty": null
      },
      {
        "id": "cd1198ca08ae179f8f5f4827fdb640269a5f0f020b5157fb9aa10b596c391414",
        "domain": "numina",
        "question": "5. Let the mass of the new alloy be 1, and the relative masses of the 1st, 2nd, and 3rd alloys in it be \\( x, y, z \\) respectively; the content of chromium in the new alloy is \\( k \\). We obtain the system of equations:\n\n$$\n\\left\\{\\begin{array}{c}\nx+y+z=1 \\\\\n0.9 x+0.3 z=0.45 \\\\\n0.4 x+0.1 y+0.5 z=k\n\\end{array}\\right.\n$$\n\nFrom the 2nd equation: \\( y=\\frac{1}{2}-\\frac{1}{3} z \\). Substituting into the 1st equation, we get: \\( x=\\frac{1}{2}-\\frac{2}{3} z \\).\n\nSubstituting into the 3rd equation: \\( k=0.25+0.2 z \\).\n\nSince \\( z \\geq 0 \\), then \\( k_{\\min }=0.25=25 \\% \\).\n\n\\( x=\\frac{1}{2}-\\frac{2}{3} z \\Leftrightarrow z=\\frac{3}{4}-\\frac{3}{2} x \\)\n\n\\( k=0.25+0.2 z=0.4-0.3 x \\).\n\nSince \\( x \\geq 0 \\), then \\( k_{\\max }=0.4=40 \\% \\).",
        "answer": "2540",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13010,
        "difficulty": null
      },
      {
        "id": "8e7c81e1d2a36ed297d5fc4d4bb751ba2c872c035ae8f5ddc536b37be659bb50",
        "domain": "numina",
        "question": "(19) Let $P$ be a moving point on the major axis of the ellipse $\\frac{x^{2}}{25}+\\frac{y^{2}}{16}=1$. A line passing through $P$ with slope $k$ intersects the ellipse at points $A$ and $B$. If $|P A|^{2}+|P B|^{2}$ depends only on $k$ and not on $P$, find the value of $k$.\n\nLet $P$ be a moving point on the major axis of the ellipse $\\frac{x^{2}}{25}+\\frac{y^{2}}{16}=1$. A line passing through $P$ with slope $k$ intersects the ellipse at points $A$ and $B$. If $|P A|^{2}+|P B|^{2}$ depends only on $k$ and not on $P$, find the value of $k$.",
        "answer": "\\\\frac{4}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122972,
        "difficulty": null
      },
      {
        "id": "65e5c843af725adfa47d0dba36cbe23ab4a2b80ebba4fd27c5f1500718e4e2a0",
        "domain": "numina",
        "question": "117. Two Ferries. Two ferries travel between two opposite banks of a river at constant speeds. Upon reaching a bank, each immediately starts moving in the opposite direction. The ferries departed from opposite banks simultaneously, met for the first time 700 meters from one of the banks, continued to their respective banks, then turned around and met again 400 meters from the other bank. Determine the width of the river in your mind.",
        "answer": "1700",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 41550,
        "difficulty": null
      },
      {
        "id": "571ed0b277cedee56ff4b7b497c145fdbd292d7edb3f23ac7a97dace8d96d63e",
        "domain": "numina",
        "question": "8. For a finite set\n$$\nA=\\left\\{a_{i} \\mid 1 \\leqslant i \\leqslant n, i \\in \\mathbf{Z}_{+}\\right\\}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {, }\n$$\n\nlet $S=\\sum_{i=1}^{n} a_{i}$, then $S$ is called the \"sum\" of set $A$, denoted as $|A|$. Given the set $P=\\{2 n-1 \\mid n=1,2, \\cdots, 10\\}$, all subsets of $P$ containing three elements are $P_{1}, P_{2}, \\cdots, P_{k}$. Then $\\sum_{i=1}^{k}\\left|P_{i}\\right|=$ $\\qquad$ .",
        "answer": "3600",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 121189,
        "difficulty": null
      },
      {
        "id": "be32f257ec235d24745039a2e9fb761857bad5852471c97a5646d2c1ed31c255",
        "domain": "numina",
        "question": "11.2. For the angles of triangle $ABC$, it is known that $\\sin A + \\cos B = \\sqrt{2}$ and $\\cos A + \\sin B = \\sqrt{2}$. Find the measure of angle $C$.",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15188,
        "difficulty": null
      },
      {
        "id": "599a5cb329cd3dc31b4d67d45278e875d3a385c94d06f9b2385cdb9428d752a2",
        "domain": "numina",
        "question": "4. Out of 43 coins arranged in a row, 8 coins are facing \"tails\" up, and 35 are facing \"heads\" up. In one step, any 20 coins can be flipped. Is it possible after a finite number of steps for all the coins to be facing \"heads\" up? In the minimum number of steps, how is this possible? Explain your answer!",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 187678,
        "difficulty": null
      },
      {
        "id": "56fab71de4fcecd560af984077cc42cbcd6e265407decc9a2eebaa92b217cb16",
        "domain": "numina",
        "question": "## 10. Four colors\n\nEach of the squares in a $4 \\times 4$ grid, as shown in the figure, must be colored with one of four colors. How many ways can this be done such that any four squares sharing a common vertex are colored with different colors?\n\n![](https://cdn.mathpix.com/cropped/2024_05_30_1fa462ef9f831ff2e2d1g-6.jpg?height=306&width=309&top_left_y=492&top_left_x=1512)\n\n## Result: $\\quad 168$",
        "answer": "168",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89903,
        "difficulty": null
      },
      {
        "id": "db67f250c4ae4b23a9778d6d4bc71a097488cc309cd47c3b981665943a7241ea",
        "domain": "numina",
        "question": "## Task 2 - 340612\n\nThe tile pattern in the figure is composed of 14 white and 10 patterned triangular tiles. One can find several squares and triangles, each made up of more than one tile.\n\nHow many such squares can be found in total?\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=274&width=262&top_left_y=1391&top_left_x=1428)\n\nThere are exactly 4 squares made up of 2 tiles each, 5 squares made up of 4 tiles each, 4 squares made up of 7 tiles each, 1 square made up of 8 tiles (see examples in Fig. a, b, c, d), making a total of 14 squares of the desired type.\na)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=105&width=117&top_left_y=1935&top_left_x=655)\nb)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=106&width=111&top_left_y=1940&top_left_x=824)\nc)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=197&width=194&top_left_y=1849&top_left_x=997)\nd)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=195&width=194&top_left_y=1850&top_left_x=1251)\ne)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=102&width=106&top_left_y=2142&top_left_x=467)\nf)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=103&width=186&top_left_y=2147&top_left_x=632)\ng)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=103&width=168&top_left_y=2144&top_left_x=887)\nh)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=188&width=174&top_left_y=2062&top_left_x=1135)\ni)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=148&width=260&top_left_y=2102&top_left_x=1366)\n\nThere are exactly 20 triangles made up of 2 tiles each, 8 triangles made up of 3 tiles each, 8 triangles made up of 4 tiles each, 4 triangles made up of 8 tiles each, 4 triangles made up of 9 tiles each (see examples in Fig. e, f, g, h, i), making a total of 44 triangles of the desired type.",
        "answer": "32",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 183429,
        "difficulty": null
      },
      {
        "id": "75412a94209bcf052e89ed156d9bd9c597036d84141b7210fc30a0f34dcaca33",
        "domain": "numina",
        "question": "2. Let $a>0$, if the solution set of the inequality $\\sqrt{x+a}+\\sqrt{x-a} \\leqslant \\sqrt{2(x+1)}$ corresponds to an interval length of $\\frac{1}{2}$ on the number line, then the value of $a$ that satisfies the condition is . $\\qquad$",
        "answer": "\\frac{3}{4}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 97090,
        "difficulty": null
      },
      {
        "id": "4f48e43541524638f3eedd6375b7f6d1baa0b15f4ad05ed3811030cded163374",
        "domain": "numina",
        "question": "5. Once Valera left home, walked to the cottage, painted 11 fence boards there, and returned home 2 hours after leaving. Another time, Valera and Olga went to the cottage together, painted 9 fence boards (without helping or hindering each other), and returned home together 3 hours after leaving. How many boards will Olga be able to paint alone if she needs to return home 1 hour after leaving? The physical abilities, diligence, and working conditions of Valera and Olga remain unchanged.\n\n(V. P. Fedorov)",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11657,
        "difficulty": null
      },
      {
        "id": "aab152f1532f75df63526ea01ad27b3f9287c33f4ede753fbe46b53d18bcf863",
        "domain": "numina",
        "question": "8. Let $g(x)=\\sqrt{x(1-x)}$ be a function defined on the interval $[0,1]$, then the area of the figure enclosed by the graph of the function $x g(x)$ and the $x$-axis is $\\qquad$ .",
        "answer": "\\frac{\\pi}{16}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 147145,
        "difficulty": null
      },
      {
        "id": "be31c57ebee36f500e7eaf5cf8ce3520734134cd0eb4a93d7e6a86d4fe5e65f6",
        "domain": "numina",
        "question": "20. Given a finite number of polygons in the plane, if for any two of them, there is a line passing through the origin that intersects both, then these polygons are said to be properly placed.\n\nFind the smallest positive integer $m$, such that for any set of properly placed polygons, it is possible to draw $m$ lines passing through the origin, so that each of these polygons intersects at least one of these $m$ lines.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 145291,
        "difficulty": null
      },
      {
        "id": "c8d9621eeb1f3b1e1a39bce601129d443637d708195a26908f630ed4f2b3ab28",
        "domain": "numina",
        "question": "Determine the gcd of all numbers of the form\n\n$$\n(a-b)(b-c)(c-d)(d-a)(a-c)(b-d)\n$$\n\nwhere $a, b, c, d$ are integers.",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 83663,
        "difficulty": null
      },
      {
        "id": "e959d909cc8e52aca757e11aa5f42a35f9d7429c36482d90ce6f38c87dc346ad",
        "domain": "numina",
        "question": "4. In the picture, there is a special playing die. The numbers on opposite sides always add up to the same sum. The numbers we don't see in the picture are prime. What number is opposite the number 14, and which number is opposite the number 18?\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_46703489c20da379fcf1g-1.jpg?height=223&width=234&top_left_y=1873&top_left_x=911)",
        "answer": "2319",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 86684,
        "difficulty": null
      },
      {
        "id": "514b144b273053ffa2485816f9720da843f0741964c7852adb8ac38a80cd5b64",
        "domain": "numina",
        "question": "8. One or two? Let's take all natural numbers from 1 to 1000000 and for each of them, calculate the sum of its digits. For all the resulting numbers, we will again find the sum of their digits. We will continue this process until all the resulting numbers are single-digit. Among the million resulting numbers, 1 and 2 will appear. Which number will be more frequent: 1 or 2?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 35662,
        "difficulty": null
      },
      {
        "id": "46d21ee49f12a88f9e219b89c99ba89d2b98c6d98533ca267de270a4eb6626a7",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nSuppose that\n\n$$\n\\frac{1}{2^{12}}+\\frac{1}{2^{11}}+\\frac{1}{2^{10}}+\\cdots+\\frac{1}{2^{t+1}}+\\frac{1}{2^{t}}=\\frac{n}{2^{12}}\n$$\n\n(The sum on the left side consists of $13-t$ terms.)\n\nWhat is the value of $n$ ?",
        "answer": "127",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194351,
        "difficulty": null
      },
      {
        "id": "8ff68115455a27066d9d49fca34c5deccc440c13dd072bce2bfa351ceef3f008",
        "domain": "numina",
        "question": "Solve the following equation:\n\n$$\n\\frac{3 x+3}{\\sqrt{x}}=4+\\frac{x+1}{\\sqrt{x^{2}-x+1}} .\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171126,
        "difficulty": null
      },
      {
        "id": "7b9d4f1e8097ddea99116158b85b023431fc764fa1f208d5bbd1315efded2018",
        "domain": "numina",
        "question": "## 42. Twins\n\nAnna and Brigitte, two little girls, are twins; they look like each other as much as two drops of water. At school, they sit at the same desk; one of them always sits on the left, and the other on the right. Each of them claims that she is Brigitte. The teacher knows from experience that, generally speaking, the girl on the right lies one time out of four, while the girl on the left lies one time out of five. But the teacher is interested in the probability that Anna is sitting on the left. Help her.",
        "answer": "\\frac{3}{7}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 42020,
        "difficulty": null
      },
      {
        "id": "c27d6b9584b7bcba16ae06a146f73c3f743aef3b51315eb7aae83479b344798f",
        "domain": "numina",
        "question": "10. In the tetrahedron $P-ABC$, $PA=BC=\\sqrt{6}$, $PB=AC=\\sqrt{8}$, $PC=AB=\\sqrt{10}$. Then the radius of the circumscribed sphere of the tetrahedron is $\\qquad$.",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 100473,
        "difficulty": null
      },
      {
        "id": "b02113bad4488f609253548eb095d0b75f64cf812ed8a3274505864914bd4061",
        "domain": "numina",
        "question": "7 Given complex numbers $Z_{1}$ and $Z_{2}$ satisfy $\\left|Z_{1}\\right|=2,\\left|Z_{2}\\right|=3$. If the angle between the vectors they correspond to is $60^{\\circ}$, then $\\left|\\frac{Z_{1}+Z_{2}}{Z_{1}-Z_{2}}\\right|=$ $\\qquad$ .",
        "answer": "\\frac{\\sqrt{133}}{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 137149,
        "difficulty": null
      },
      {
        "id": "2fcd8783467ecab10a8563c4d0c089845440e72c450d23c16c82992613581446",
        "domain": "numina",
        "question": "In Anchuria, a unified state exam is taking place. The probability of guessing the correct answer to each question on the exam is 0.25. In 2011, to obtain a certificate, one needed to answer three questions correctly out of 20. In 2012, the School Administration of Anchuria decided that three questions were too few. Now, one needs to answer six questions correctly out of 40. The question is, if one knows nothing and simply guesses the answers, in which year is the probability of obtaining an Anchurian certificate higher - in 2011 or in 2012?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57688,
        "difficulty": null
      },
      {
        "id": "2315322b87d76a0e2b79db8028e90d61a72d0f9b5be2dcd2f470545e354bc403",
        "domain": "numina",
        "question": "$9.17 C_{x}^{x-1}+C_{x}^{x-2}+C_{x}^{x-3}+\\ldots+C_{x}^{x-9}+C_{x}^{x-10}=1023$.",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 49874,
        "difficulty": null
      },
      {
        "id": "21e188420fa5f192d7a3cfae850d3cdcc097cc5a1f4617e75d3ae0b409ef2875",
        "domain": "numina",
        "question": "57. The number $123456789(10)(11)(12)(13)(14)$ is written in the base-15 numeral system, i.e., this number is equal to\n\n(14) $+(13) \\cdot 15+(12) \\cdot 15^{2}+(11) \\cdot 15^{3}+\\ldots+2 \\cdot 15^{12}+15^{13}$. What remainder does it give when divided by 7?",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 29329,
        "difficulty": null
      },
      {
        "id": "fcc49e182fefcaa17d46d51b4cf7acc444c0543692a271066163943673a55e5d",
        "domain": "numina",
        "question": "6. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies:\n$$\na_{1}=1, a_{n+1}=\\frac{(n+1) a_{n}}{2 n+a_{n}}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen $\\sum_{k=1}^{2021} \\frac{k}{a_{k}}=$ . $\\qquad$",
        "answer": "2^{2022}-2023",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133538,
        "difficulty": null
      },
      {
        "id": "951346b5a1b7ac3673de9132e2e1132a1dd797a004fa521d7edfd3cf77dab9ac",
        "domain": "numina",
        "question": "5. The numbers from 1 to 8 are arranged at the vertices of a cube such that the sum of the numbers in any three vertices lying on the same face is at least 10. What is the smallest possible sum of the numbers at the vertices of one face?",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 5852,
        "difficulty": null
      },
      {
        "id": "4b584362c9756ee0daafafac24bdf334ee54e2164662ec68797fd746cdde54e6",
        "domain": "numina",
        "question": "16. Arrange $a, b, c, d, e, f$ such that the letter $b$ always appears before the letter $e$. How many such arrangements are there?",
        "answer": "360",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 100316,
        "difficulty": null
      },
      {
        "id": "c9561978b2d9df486c1d730af895b160075e861151da873025120fc92502ed80",
        "domain": "numina",
        "question": "8th Mexico 1994 Problem B1 A capricious mathematician writes a book with pages numbered from 2 to 400. The pages are to be read in the following order. Take the last unread page (400), then read (in the usual order) all pages which are not relatively prime to it and which have not been read before. Repeat until all pages are read. So, the order would be 2, 4, 5, ... , 400, 3, 7, 9, ... , 399, ... . What is the last page to be read?",
        "answer": "37",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 149129,
        "difficulty": null
      },
      {
        "id": "97e725434e1ed9491bddac4b51a71878230e1f5b09d7f4597874516a5e76f21d",
        "domain": "numina",
        "question": "10. Determine the maximum value attained by\n$$\n\\frac{x^{4}-x^{2}}{x^{6}+2 x^{3}-1}\n$$\nover real numbers $x>1$.",
        "answer": "\\frac{1}{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152377,
        "difficulty": null
      },
      {
        "id": "5acf4840c4a2881f4d9af7f757da64c9743a7835c31d33d200bebabbc38c9ee1",
        "domain": "numina",
        "question": "8. (10 points) Divide a cubic wooden block with an edge length of 1 meter into small cubic building blocks with an edge length of 1 centimeter. Imagine that Sun Wukong uses his divine power to stack all the small building blocks one by one, forming a rectangular \"magic rod\" pointing to the sky. Knowing that the altitude of Mount Everest is 8844 meters, then the height of the \"magic rod\" exceeds the altitude of Mount Everest by meters.",
        "answer": "1156",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102311,
        "difficulty": null
      },
      {
        "id": "d91fb7a924bb888fdd2d8db351cb380a5aca31df741565ef9063a8f622accd84",
        "domain": "numina",
        "question": "G47 (23-5, Netherlands) Given that $A C, C E$ are two diagonals of a regular hexagon $A B C D E F$, and points $M, N$ internally divide $A C, C E$ respectively, such that $\\frac{A M}{A C}=\\frac{C N}{C E}=r$. If $B, M, N$ are collinear, find the value of $r$.\n\n---\n\nThe problem is to find the value of $r$ given that points $M$ and $N$ divide the diagonals $A C$ and $C E$ of a regular hexagon $A B C D E F$ in the ratio $r$, and that points $B, M, N$ are collinear.",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126852,
        "difficulty": null
      },
      {
        "id": "8b49c5a1cb7bbe94c6baa62426022060aadcd333a55dc1d5bfeadc5e3a08647b",
        "domain": "numina",
        "question": "39th Putnam 1978 Problem B1 A convex octagon inscribed in a circle has 4 consecutive sides length 3 and the remaining sides length 2. Find its area.",
        "answer": "13+12\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147760,
        "difficulty": null
      },
      {
        "id": "6562f97da68903309d873c47d46a474fbc4308f6abbfe57f6a567b62ec4c9873",
        "domain": "numina",
        "question": "11.3. How many points are there on the hyperbola $y=\\frac{2013}{x}$ such that the tangent at these points intersects both coordinate axes at points with integer coordinates",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 22214,
        "difficulty": null
      },
      {
        "id": "8ce9a6e4c85e3afae9be2ffc7cc60081ac8df908b3ff8d08274174cdd5fa7c35",
        "domain": "numina",
        "question": "## Problem 2\n\nFind the smallest positive integer $k$ for which there exist a colouring of the positive integers $\\mathbb{Z}_{>0}$ with $k$ colours and a function $f: \\mathbb{Z}_{>0} \\rightarrow \\mathbb{Z}_{>0}$ with the following two properties:\n\n(i) For all positive integers $m, n$ of the same colour, $f(m+n)=f(m)+f(n)$.\n\n(ii) There are positive integers $m, n$ such that $f(m+n) \\neq f(m)+f(n)$.\n\nIn a colouring of $\\mathbb{Z}_{>0}$ with $k$ colours, every integer is coloured in exactly one of the $k$ colours. In both (i) and (ii) the positive integers $m, n$ are not necessarily different.\n\nMerlijn Staps, the Netherlands",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 161624,
        "difficulty": null
      },
      {
        "id": "07e2a73127c25deb367ceeba0bb176bed23cca63959eea09b87606bf521e4a8d",
        "domain": "numina",
        "question": "7.4. How to cut a $5 \\times 5$ square with straight lines so that the resulting pieces can be used to form 50 equal squares? It is not allowed to leave unused pieces or overlap them.",
        "answer": "50",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14903,
        "difficulty": null
      },
      {
        "id": "138cbf6e49bd74c3a0cf9ba574c5d2965b1730718d090555d52a1abd9a29aa4a",
        "domain": "numina",
        "question": "4. In the Olympionov family, it is a tradition to especially celebrate the day when a person turns as many years old as the sum of the digits of their birth year. Kolya Olympionov had such a celebration in 2013, and Tolya Olympionov had one in 2014. Who is older and by how many years?",
        "answer": "4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 11781,
        "difficulty": null
      },
      {
        "id": "a98a3383ce0fb344c9ea6916cbeb56c326443786bdabc81889e6309ec9e88160",
        "domain": "numina",
        "question": "1.97 Find the largest integer $n$ such that $\\frac{(n-2)^{2}(n+1)}{2 n-1}$ is an integer.\n(17th American Invitational Mathematics Examination, 1999)",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109489,
        "difficulty": null
      },
      {
        "id": "4b1b8363c27af4d0ba34cce77ed8126977cee1a15a5e3abcbc93fb7d3551c8ec",
        "domain": "numina",
        "question": "1. Calculate: $3752 \\div(39 \\times 2)+5030 \\div(39 \\times 10)=$",
        "answer": "61",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129628,
        "difficulty": null
      },
      {
        "id": "4aa8101ebfe276ae48917bb38d676ce1f4817bf0385086ffbff7003a496c92df",
        "domain": "numina",
        "question": "10. As shown in the figure, two squares $\\mathrm{ABEG}, \\mathrm{GECD}$, point $\\mathrm{H}$ is the midpoint of $\\mathrm{GE}$, $\\frac{D F}{D C}=\\frac{1}{3}$. Connect $\\mathrm{DH}$, $\\mathrm{CH}$, $\\mathrm{AF}$, $\\mathrm{BF}$. The area of square $\\mathrm{ABEG}$ is $\\mathrm{m}$ square centimeters, and the area of the shaded part is $\\mathrm{n}$ square centimeters. Given that $\\mathrm{m}$ and $\\mathrm{n}$ are positive integers, and $\\mathrm{m}$ has 9 divisors, then the side length of square $\\mathrm{ABEG}$ is $\\qquad$ centimeters.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 141333,
        "difficulty": null
      },
      {
        "id": "d8d6b55d256c4398f731154c0f6996b40562dab7b8629e4996ea407e340f3d8e",
        "domain": "numina",
        "question": "12. In the tetrahedron $ABCD$, there is a point $O$ inside such that the lines $AO, BO, CO, DO$ intersect the faces $BCD, ACD, ABD, ABC$ at points $A_1, B_1, C_1, D_1$ respectively, and $\\frac{AO}{OA_1}=\\frac{BO}{OB_1}=\\frac{CO}{OC_1}=\\frac{DO}{OD_1}=k$. Then $k=$ $\\qquad$ .",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 112572,
        "difficulty": null
      },
      {
        "id": "495c8d7bcdd3212f7ba795dfa3500c225f97acb25813e9877145eb498de20666",
        "domain": "numina",
        "question": "Example 8 Given an integer $n \\geqslant 2$, try to determine the smallest constant $c$, such that for any non-negative real numbers $x_{i}(1 \\leqslant i \\leqslant n)$, we have $\\sum_{1 \\leqslant i<j \\leqslant n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4}$. And for this constant $c$, determine the necessary and sufficient conditions for equality to hold.",
        "answer": "\\frac{1}{8}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 112234,
        "difficulty": null
      },
      {
        "id": "a32f90633dfe54b3c8071629f9f8b72b9c4647207158c271be71fa1bbb8d9d1a",
        "domain": "numina",
        "question": "11.4 If $\\sqrt{3-2 \\sqrt{2}}=\\sqrt{c}-\\sqrt{d}$, find the value of $d$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152695,
        "difficulty": null
      },
      {
        "id": "e9ebed0aaad0d0147096481a81094ca285687bd215e974bd994ba6df0d33829a",
        "domain": "numina",
        "question": "11.9. In a class, there are $m$ students. During September, each of them went to the swimming pool several times; no one went twice on the same day. On October 1st, it turned out that all the numbers of swimming pool visits by the students were different. Moreover, for any two of them, there was definitely a day when the first one was at the pool and the second one was not, and a day when, conversely, the second one was at the pool and the first one was not. Find the maximum possible value of $m$. (September has 30 days.)",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12970,
        "difficulty": null
      },
      {
        "id": "d4a6ff2fa23600193aa10d4689b8dfbf34268732dcd2a875605ad2b82e9954b0",
        "domain": "numina",
        "question": "1. (2 points) Does there exist a four-digit natural number with the sum of its digits being 21, which is divisible by $14 ?$",
        "answer": "6384",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 21181,
        "difficulty": null
      },
      {
        "id": "26b2d164af6c01e8dc3f570616683dfe4a6141edccdcc2bc7c1cf90a04c49b47",
        "domain": "numina",
        "question": "## Task B-3.4.\n\nA circle with a diameter of $5 \\mathrm{~cm}$ is inscribed in an isosceles trapezoid with an area of $36 \\mathrm{~cm}^{2}$. Determine the perimeter of the trapezoid.",
        "answer": "28.8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89346,
        "difficulty": null
      },
      {
        "id": "b9b81a277964b6bdd38df6e5f7133442653105f3e8c93644a42ace4317ac51b8",
        "domain": "numina",
        "question": "In a certain right-angled triangle, the sum of the hypotenuse and one of the legs is $c+b=14 \\text{~cm}, \\operatorname{tg} \\frac{\\alpha}{2}=\\frac{1}{2}$, where $\\alpha$ is the angle opposite the leg $a$. Calculate the lengths of the sides of the triangle without using trigonometric tables.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173018,
        "difficulty": null
      },
      {
        "id": "f866907a1848c71ddc892cd653a0b942d280123d10c88d4b55265e3dcc8b407c",
        "domain": "numina",
        "question": "3. What can the value of the expression $p^{4}-3 p^{3}-5 p^{2}+16 p+2015$ be if $p$ is a root of the equation $x^{3}-5 x+1=0$?\n\nAnswer: 2018",
        "answer": "2018",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13027,
        "difficulty": null
      },
      {
        "id": "66a7cbdb698e13ccfe46f9858462780b43f1e22b28d34a9031d0b30e07b77fa1",
        "domain": "numina",
        "question": "19. (7 points) When Wang Ming is 300 meters away from home after school, his sister and the puppy run towards him at the same time. Wang Ming and his sister both run at a speed of 50 meters per minute, while the puppy runs at a speed of 200 meters per minute. The puppy meets Wang Ming and then keeps running back and forth between Wang Ming and his sister at the same speed. When Wang Ming and his sister are 10 meters apart, how many meters has the puppy run in total?",
        "answer": "580",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 122411,
        "difficulty": null
      },
      {
        "id": "92b0e14ac7088515ba44b349b3fe07d663ebb8354dfdc2236cbbb7fba3b8c8d6",
        "domain": "numina",
        "question": "4. Given a tetrahedron $S-ABC$ with the base being an isosceles right triangle with $AB$ as the hypotenuse, $SA=SB=SC=2, AB=$ 2, and suppose points $S, A, B, C$ all lie on a sphere with center $O$. Then the distance from point $O$ to the plane $ABC$ is $\\qquad$",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126421,
        "difficulty": null
      },
      {
        "id": "0f86c8e27478f7d5d01c76f56276f0f0434efd0fd10b6aa9d651d396efa85372",
        "domain": "numina",
        "question": "## Task 5\n\nPeter calculates the sum of three numbers $a, b$, and $c$.\n\nThe number $a$ is 100 greater than the difference between the numbers 50700 and 30200.\n\nThe number $b$ is the successor of seven times 583.\n\nThe number $c$ is the third part of the predecessor of 2101.\n\nGive the numbers $a, b$, and $c$ and their sum.",
        "answer": "25382",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186853,
        "difficulty": null
      },
      {
        "id": "705300f91cb493f3f3f554afdc787630b95f0bcd2a55c5ffcfe538f0807d6b12",
        "domain": "numina",
        "question": "12. Nick and Tim are playing a game, in which there are no ties in each round. In a round of the game, the losing side must take at least one candy, and the winning side takes more candies than the losing side. They also agreed in advance on the number of candies (two given positive integers) that the losing and winning sides should take in each round. After several rounds, Nick has a total of 30 candies, and Tim has a total of 25 candies. It is known that Tim won 2 rounds. Question: How many candies should the winning side take in each round?\n(2004 Slovenian Olympiad Problem)",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 139230,
        "difficulty": null
      },
      {
        "id": "673f2929dac1b97e6030bad69e29b13cafeb7cca794c93456734a4ac644e168f",
        "domain": "numina",
        "question": "1. (3 points) Xiaohui started from point $A$ and walked 15 meters west to reach point $B$, then walked 23 meters east from point $B$ to reach point $C$. The distance from point $C$ to point $A$ is $\\qquad$ meters.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 96459,
        "difficulty": null
      },
      {
        "id": "1f7ab4ba096c6efe66ff5c090ca57ec7c4abd6af3c2d28aa98200da2b062e027",
        "domain": "numina",
        "question": "3. Schoolchildren were given 7 sheets of paper and asked to cut some of them into 7 pieces. The resulting pieces of paper were mixed and they were asked again to cut some of them into 7 pieces. This was repeated several times. How many pieces of paper need to be cut so that in the end there are 331 pieces.",
        "answer": "54",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 21895,
        "difficulty": null
      },
      {
        "id": "f5c01a521e4b8e317362039d20c51a67a6efb3babd2c15c13ac937f6dcb633df",
        "domain": "numina",
        "question": "Russian problems Problem 92 ABCD is a unit square. One vertex of a rhombus lies on side AB, another on side BC, and a third on side AD. Find the area of the set of all possible locations for the fourth vertex of the rhombus. Solution",
        "answer": "2\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147589,
        "difficulty": null
      },
      {
        "id": "0ccac4b7ff1166f36a635ce809762671497c9ff09d13dc3f5edf8b335fb25244",
        "domain": "numina",
        "question": "Let $n$ be a two-digit number written in the decimal system, and let $s$ be the sum of the squares of its digits. What is the smallest and largest value of $n-s$?",
        "answer": "25-63",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171830,
        "difficulty": null
      },
      {
        "id": "7d83c4c9654ff3a6fe56490d06a69481f5615cc549d351e674c10ae75fdd24ea",
        "domain": "numina",
        "question": "18. A positive integer is said to be 'good' if each digit is 1 or 2 and there is neither four consecutive 1 's nor three consecutive 2 's. Let $a_{n}$ denote the number of $n$-digit positive integers that are 'good'. Find the value of $\\frac{a_{10}-a_{8}-a_{5}}{a_{7}+a_{6}}$.\n(2 marks)\n若某正整數的每個數字均為 1 或 2 , 且當中既沒有四個連續的 $\\ulcorner 1$ 亦沒有三個連續的 22 , 便稱為 「好數」。設 $a_{n}$ 表示 $n$ 位「好數」的數目。求 $\\frac{a_{10}-a_{8}-a_{5}}{a_{7}+a_{6}}$ 的值。",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153669,
        "difficulty": null
      },
      {
        "id": "4aee227bded8aee7be94fcd34b5b3d90f035f6606c4cea6a2125adbca7e1596a",
        "domain": "numina",
        "question": "585. Find the last non-zero digit of the product of all natural numbers from 1 to 40.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 45212,
        "difficulty": null
      },
      {
        "id": "4020a5be8c0ba14dd2f933c30233e1df3f77794742c57193e14fb718ffb498d2",
        "domain": "numina",
        "question": "9.2. a) Given the quadratic equation $x^{2}-9 x-10=0$. Let $a$ be its smallest root. Find $a^{4}-909 a$. b) For the quadratic equation $x^{2}-9 x+10=0$, where $b$ is the smallest root, find $b^{4}-549 b$.",
        "answer": "910",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 21996,
        "difficulty": null
      },
      {
        "id": "890db4ebda581bc72c6f86168626e49249358e6789e180b2d3328bce5beb7ddb",
        "domain": "numina",
        "question": "8. For a four-digit number $\\overline{a b c d}(1 \\leqslant a \\leqslant 9,0 \\leqslant b, c, d \\leqslant 9)$, if $a>b, bd$, then $\\overline{a b c d}$ is called a $P$ class number; if $ac, c<d$, then $\\overline{a b c d}$ is called a $Q$ class number. Let $N(P)$ and $N(Q)$ represent the number of $P$ class numbers and $Q$ class numbers, respectively, then the value of $N(P)-N(Q)$ is $\\qquad$.",
        "answer": "285",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 118089,
        "difficulty": null
      },
      {
        "id": "da1dd869211e2e30e0d6fc0a0ad5b816b43fde332a13421c76aec0451b4c7458",
        "domain": "numina",
        "question": "5. If $a>b>c>d$, and $\\frac{1}{a-b}+\\frac{1}{b-c}+\\frac{1}{c-d} \\geq \\frac{n}{a-d}$, then the maximum value of the integer $n$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135815,
        "difficulty": null
      },
      {
        "id": "786ed2cc27e4c46a61c3a8dda90225554960b71d441c9d9e8a84fdd45737c996",
        "domain": "numina",
        "question": "6. A circle of radius 2, inscribed in triangle $A B C$, touches side $B C$ at point $D$. A circle of radius 4 touches the extensions of sides $A B$ and $A C$, as well as side $B C$ at point $E$. Find the area of triangle $A B C$, if the measure of angle $A C B$ is $120^{\\circ}$.",
        "answer": "\\frac{56}{\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10189,
        "difficulty": null
      },
      {
        "id": "bed9310f06228d0b05aea446df7490876f990a25e90d3d1dc562f6b1f8561750",
        "domain": "numina",
        "question": "7. (4 points) With the number written on the board, one of the following operations is allowed:\n1) If there is a digit in the original number that is not equal to 9 and has two neighboring digits greater than 0, you can increase this digit by 1, and decrease the neighboring digits by 1.\n2) Subtract 1 from any non-zero digit except the last one, and add 3 to the next digit.\n3) Decrease any sufficiently large digit by 7.\n\nIf, as a result of any of these operations, there are zeros in the number on one or more of the leading positions, they are automatically discarded.\n\nInitially, the number consisting of one hundred nines was written on the board. In the end, a single-digit number remained. Which one exactly?",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 20925,
        "difficulty": null
      },
      {
        "id": "2254e9ceb44ab893d3da4c06395e81c5b7bcfdac91526b9d35f06bea74e001aa",
        "domain": "numina",
        "question": "6. (1995 National High School Mathematics Competition) Find all real numbers $p$ such that the cubic equation $5 x^{3}-5(p+1) x^{2}+(71 p-1) x+1=66 p$ has three roots that are all natural numbers.",
        "answer": "76",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139418,
        "difficulty": null
      },
      {
        "id": "6d67701ca80f96cf48618ebbc6e28729209b6381fc26ee34b0caa7101a3413b8",
        "domain": "numina",
        "question": "Example 12. A factory sent 5000 good-quality items to a base. The probability that an item will be damaged during transportation is 0.0002. What is the probability that 3 defective items will arrive at the base?\n\n$\\mathrm{Pe} \\mathrm{sh} \\mathbf{e n i e . ~ F r o m ~ t h e ~ c o n d i t i o n ~ i t ~ f o l l o w s ~ t h a t ~} n=5000, p=0.0002, \\text{ and } np=5000 \\cdot 0.0002=1$. According to formula (3.3.1) we have\n\n$$\nP_{5000}(3)=\\frac{1^{3}}{3!} e^{-1}=\\frac{1}{6 e} \\approx 0.0613\n$$\n\nExample 13. A radio apparatus consists of 1000 electrical elements. The probability of failure of one element in one year of operation is 0.001 and is independent of the state of other elements. What is the probability of the failure of two elements? What is the probability of the failure of at least two elements in a year?",
        "answer": "0",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 33076,
        "difficulty": null
      },
      {
        "id": "be13921feb18b7ac78dcee9f35019d33b6611724ae1ce2e2286baf39a4dcac39",
        "domain": "numina",
        "question": "## Task B-1.7.\n\nIn a right-angled triangle $ABC$, the right angle is at vertex $C$, $|AB|=8$, and $\\angle ABC=60^{\\circ}$. The triangle $ABC$ is rotated around vertex $C$ by $30^{\\circ}$ counterclockwise, mapping it to triangle $A'B'C$. What is the area of the common part of triangles $ABC$ and $A'B'C$?",
        "answer": "24-10\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87257,
        "difficulty": null
      },
      {
        "id": "fe21d63419bf21778b73ece0fe93356e9fd232a25c9da8830d193daf55529438",
        "domain": "numina",
        "question": "36. The number of students in grades V-VI of one of the schools is expressed by a three-digit number. If you find the product of the digits of this number, and then the product of the digits of the resulting number, then all these three numbers can be written as:\n\n$$\n\\triangle \\bigcirc O ; \\triangle \\square ; \\square\n$$\n\n(Identical shapes correspond to identical digits.) How many students are there in grades V-VI in the school?",
        "answer": "144",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39305,
        "difficulty": null
      },
      {
        "id": "99be870f2f0a100e13d956487276dcb9cda137a6fd26fc6c7db935eab1be20a3",
        "domain": "numina",
        "question": "Problem 15. Solve the equation\n\n$$\n\\frac{5}{\\sqrt{1-x^{2}}}=\\frac{1}{x}+\\frac{85}{12}\n$$\n\n104",
        "answer": "0.8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 37570,
        "difficulty": null
      },
      {
        "id": "3eb9c095260fb3b0cbb704b8fe095435956287c44274376902a52496d2742f0f",
        "domain": "numina",
        "question": "Do there exist positive integers $x, y, z$ such that $28 x+30 y+31 z=365$ ? What is the largest positive integer $n$ such that the equation $28 x+30 y+31 z=n$ cannot be solved in positive integers $x, y, z$ ?",
        "answer": "370",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 90562,
        "difficulty": null
      },
      {
        "id": "2563a99b8f6c5fa44b80cdd12642387db829746e67853ed34eda02299f3dcae9",
        "domain": "numina",
        "question": "## [\n\nIn an isosceles trapezoid with bases 1 and 4, two circles are placed, each touching the other circle, the two lateral sides, and one of the bases. Find the area of the trapezoid.\n\n#",
        "answer": "\\frac{15\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58414,
        "difficulty": null
      },
      {
        "id": "8e51664ecdc4d472ff1cb84b588e3b720dfe9958ca2fe22514303e7ffc08bcad",
        "domain": "numina",
        "question": "Example 12 Let $X=\\{1,2, \\cdots, 2001\\}$. Find the smallest positive integer $m$, such that for any $m$-element subset $W$ of $X$, there exist $u, v \\in W$ (where $u$ and $v$ can be the same), such that $u+v$ is a power of 2.\n(2001 China Mathematical Olympiad)",
        "answer": "999",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96772,
        "difficulty": null
      },
      {
        "id": "bf8e411e44683a7c7309f372a764faf65e2ea6cb727c63d7a6feeb9fff0d70f2",
        "domain": "numina",
        "question": "B2. Alice places a coin, heads up, on a table then turns off the light and leaves the room. Bill enters the room with 2 coins and flips them onto the table and leaves. Carl enters the room, in the dark, and removes a coin at random. Alice reenters the room, turns on the light and notices that both coins are heads. What is the probability that the coin Carl removed was also heads?",
        "answer": "\\frac{3}{5}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 139613,
        "difficulty": null
      },
      {
        "id": "7c5ba9e0dd8f3cd4a8e0c3e5f6cda91aa7283c47b756a164a2f49b351c2c8e99",
        "domain": "numina",
        "question": "Exercise 14. Let $x$ be a rational number such that $\\sqrt{x}+\\sqrt{x+13}=\\sqrt{4 x+17}$. We write $x$ in the form $x=\\frac{a}{b}$ where $a$ and $b$ are natural numbers with no common divisors. Determine the value of $a+b$.\n\n---\n\nThe translation is provided as requested, maintaining the original formatting and structure.",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 78310,
        "difficulty": null
      },
      {
        "id": "7efca330d7fe31bf6184496597985d5ee56050f05dd86743bcc2a373a416450a",
        "domain": "numina",
        "question": "11.4. A set of composite numbers from the set $\\{1,2,3,4, \\ldots, 2016\\}$ is called good if any two numbers in this set do not have common divisors (other than 1). What is the maximum number of numbers that a good set can have?",
        "answer": "14",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13724,
        "difficulty": null
      },
      {
        "id": "b48c1c1a268927fd496cdefb0218dc5593379ec6c9ea8e96e21f71ad8f839439",
        "domain": "numina",
        "question": "[ Work Problems.]\n\nThree workers are digging a pit. They work in turns, and each of them works as long as it takes the other two to dig half of the pit. Working this way, they dug the pit. How many times faster would three workers dig the same pit if they worked simultaneously?",
        "answer": "2.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70056,
        "difficulty": null
      },
      {
        "id": "006b060283a0b709b7fd27bca85b6c05b4583cc3b27a66b4e7f5b3e2af8ddcab",
        "domain": "numina",
        "question": "10.218. A line parallel to the bases of a trapezoid passes through the point of intersection of its diagonals. Find the length of the segment of this line, enclosed between the lateral sides of the trapezoid, if the bases of the trapezoid are 4 and 12 cm.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 48827,
        "difficulty": null
      },
      {
        "id": "ddfb81eef48305c2b5fe9d6c3b50a496f6929210b05c10790014f48b05b11913",
        "domain": "numina",
        "question": "For the elements of the sequence $a_{n}$, it holds that $a_{1}=1337$, and furthermore, that $a_{2 n+1}=a_{2 n}=n-a_{n}$ for every positive integer $n$. Determine the value of $a_{2004}$.",
        "answer": "2004",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170270,
        "difficulty": null
      },
      {
        "id": "2d0e22b892be40a5117703524e80f7be57a1362e855c659394d1645ce65e61e7",
        "domain": "numina",
        "question": "Robots Robert and Hubert assemble and disassemble coffee grinders. Each of them assembles a coffee grinder four times faster than they can disassemble it. When they came to the workshop in the morning, several coffee grinders were already assembled there.\n\nAt 7:00, Hubert started assembling and Robert started disassembling, and precisely at 12:00, Hubert finished assembling a coffee grinder and Robert finished disassembling another one. In total, 70 coffee grinders were added during this shift.\n\nAt 13:00, Robert started assembling and Hubert started disassembling, and precisely at 22:00, Robert finished assembling the last coffee grinder and Hubert finished disassembling another one. In total, 36 coffee grinders were added during this shift.\n\nHow long would it take to assemble 360 coffee grinders if Robert and Hubert were assembling together? (K. Pazourek)\n\nHint. How many coffee grinders will be added per hour in each shift?",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76199,
        "difficulty": null
      },
      {
        "id": "995b2e5dc454db2f1db2ad7e19c2e107a1d495dbf869e32478f7ecd0d5e6bffd",
        "domain": "numina",
        "question": "1. (2 points) In trapezoid $A B C D$ with bases $A D=16$ and $B C=10$, the circles constructed on sides $A B, B C$ and $C D$ as diameters intersect at one point. The length of diagonal $A C$ is 10. Find the length of $B D$.",
        "answer": "24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21105,
        "difficulty": null
      },
      {
        "id": "8425dfd271810f167c9db98f1c6e2a77a011c52618b97a4fd0785d6d634fc4b0",
        "domain": "numina",
        "question": "If the Earth were to suddenly stop its orbit around the Sun and fall into the Sun, how long would it take for the Earth to reach the Sun?",
        "answer": "64.5",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 166550,
        "difficulty": null
      },
      {
        "id": "d011420edc229762e76082a34fea80229844258ac5c75a7861f609b336c72285",
        "domain": "numina",
        "question": "For positive real numbers $x$ and $y$, the following holds:\n\n$$\n2 \\log (x-2 y)=\\log x+\\log y\n$$\n\nWhat is $\\frac{x}{y} ?$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162664,
        "difficulty": null
      },
      {
        "id": "3cd8d64499bbb3c894327dac2459e18989f0d66a542b5b3b7e9943dc7f637ca9",
        "domain": "numina",
        "question": "Example 1 As shown in Figure 13-4, in $\\triangle A B C$, $O$ is the circumcenter of $\\triangle A B C$, and $I$ is its incenter. If $\\angle B O C=$ $\\angle B I C$, find $\\angle A$.\n\n保留了原文的格式和换行。",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143172,
        "difficulty": null
      },
      {
        "id": "9bfa6874b58ced803f91d319c0b2a7827aa4b9106783bdbd035c405bdf51e16f",
        "domain": "numina",
        "question": "Problem 7.4.1. Brothers Lёsha and Sasha decided to get to the skate park. They left at the same time, but Lёsha walked with the skateboard in his hands, while Sasha rode on the skateboard. It is known that Sasha rides the skateboard 3 times faster than Lёsha walks with the skateboard. After some time, they simultaneously switched their modes of transportation: Lёsha started riding the skateboard, and Sasha started walking. As a result, each of their speeds changed by a factor of 2: Lёsha's speed increased, and Sasha's speed decreased. It turned out that they arrived at the skate park at the same time. How many meters did Sasha ride on the skateboard, if the distance from home to the skate park is 3300 meters?",
        "answer": "1100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16021,
        "difficulty": null
      },
      {
        "id": "6e25a6d69ded7b08ec4b71ddae8296927dcbdf030df8a22195ab59fffaa66c6c",
        "domain": "numina",
        "question": "4. Given real numbers $x, y$ satisfy\n$$\n2 x=\\ln (x+y-1)+\\ln (x-y-1)+4 \\text {. }\n$$\n\nThen the value of $2015 x^{2}+2016 y^{3}$ is $\\qquad$",
        "answer": "8060",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135746,
        "difficulty": null
      },
      {
        "id": "a480998aeab21bb0a9752c782df8e53117b562862dd940ae91e2dbdcc3bbec98",
        "domain": "numina",
        "question": "Someone says that seven times their birth year, when divided by 13, leaves a remainder of 11, and thirteen times their birth year, when divided by 11, leaves a remainder of 7. In which year of their life will this person be in 1954?",
        "answer": "1868",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 172318,
        "difficulty": null
      },
      {
        "id": "71000a5dffb0dd2b441bc06808e890dcf361ac69522ae6e0ec123da465247e56",
        "domain": "numina",
        "question": "Example 3. Solve the equation\n\n$$\n5^{2 x+1}-3 \\cdot 5^{2 x-1}=550\n$$",
        "answer": "\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 40524,
        "difficulty": null
      },
      {
        "id": "a004356550ac8ffea31e6e546534ddd3fe6e9127e20d1f7212b1f0a4ab4e1e01",
        "domain": "numina",
        "question": "1. In triangle $A B C, A B=6, B C=10$, and $C A=14$. If $D, E$, and $F$ are the midpoints of sides $B C$, $C A$, and $A B$, respectively, find $A D^{2}+B E^{2}+C F^{2}$.",
        "answer": "249",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 154627,
        "difficulty": null
      },
      {
        "id": "edc417256888b78cedd83b01391cf0f87f859e7ab275814ecfea03caf2310f0b",
        "domain": "numina",
        "question": "GS. 2 If $R$ is the remainder of $1^{6}+2^{6}+3^{6}+4^{6}+5^{6}+6^{6}$ divided by 7 , find the value of $R$.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153163,
        "difficulty": null
      },
      {
        "id": "9073adca5dff2322774e1e387ee3877e005442fcb49e8580c59c1ccdbd514acf",
        "domain": "numina",
        "question": "6.58. $\\lim _{x \\rightarrow+0} x^{x}$.",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33410,
        "difficulty": null
      },
      {
        "id": "e99810652f9cce047dea0d19d6610cba93adf53666a2cffd28bf77ceca04f556",
        "domain": "numina",
        "question": "225. Fruit Garden. The gardener decided to create a new fruit garden. He planted young trees in rows so that a square was formed. In doing so, he had 146 extra seedlings left. But to increase the square by adding an extra row, the gardener had to buy 31 more trees.\n\nHow many trees are there in the garden at the end of the work?",
        "answer": "7921",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41141,
        "difficulty": null
      },
      {
        "id": "ddf2142bb7fc7d22dd57d003d2481273d2ee24977d0b9c275efdfcb4a0539f0e",
        "domain": "numina",
        "question": "4. Let $X=\\{1,2,3, \\ldots, 12\\}$. Find the the number of pairs $\\{A, B\\}$ such that $A \\subseteq X$, $B \\subseteq X, A \\neq B$ and $A \\cap B=\\{2,3,5,7,8\\}$.",
        "answer": "3^{7}-1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 180344,
        "difficulty": null
      },
      {
        "id": "1e9d646fdbc265e42375629d657d113f9d3f413a1a130b73585335c435d4109e",
        "domain": "numina",
        "question": "In how many ways can we remove 8 letters from the 16 letters in Figure 1 so that two are missing from each row and each column?\n\n| $A$ | $B$ | $C$ | $D$ |\n| :---: | :---: | :---: | :---: |\n| $E$ | $F$ | $G$ | $H$ |\n| $I$ | $J$ | $K$ | $L$ |\n| $M$ | $N$ | $O$ | $P$ |",
        "answer": "90",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 166306,
        "difficulty": null
      },
      {
        "id": "fda507c36a19ffd8a2553bc2f72894654bc748b2f261519b6b96223330ac98fb",
        "domain": "numina",
        "question": "8. It is given that $\\sqrt{a}-\\sqrt{b}=20$, where $a$ and $b$ are real numbers. Find the maximum possible value of $a-5 b$.",
        "answer": "500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160524,
        "difficulty": null
      },
      {
        "id": "658ea1372f0b68a2510b1bd71afa8db7a0661c705f2b260207b252c082182bb9",
        "domain": "numina",
        "question": "1. The solution to the equation $\\frac{16}{3} x^{4}+\\frac{1}{6 x^{2}}=\\sin \\pi x$ is",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94859,
        "difficulty": null
      },
      {
        "id": "34bc41b77c742bb634351ffb2794cbf55516ddd096b85406cc5e6e40c14fc90c",
        "domain": "numina",
        "question": "1. [5] Solve for $x$ in the equation $20 \\cdot 14+x=20+14 \\cdot x$.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151541,
        "difficulty": null
      },
      {
        "id": "aa97bdc0dbcebb51b271c2e08eaf83825b9994a8371a55e06c4b89dc48636644",
        "domain": "numina",
        "question": "5. What is the maximum number of rooks that can be placed on the cells of a $300 \\times 300$ board so that each rook attacks no more than one other rook? (A rook attacks all cells it can reach according to chess rules, without passing through other pieces.)\n\n#",
        "answer": "400",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7863,
        "difficulty": null
      },
      {
        "id": "15c00e7ae506a7bb8cbc04869e1d88a825c844a9f0e34b7c0a237cab9a12893a",
        "domain": "numina",
        "question": "6.1. On 19 cards, the numbers $15,16,17, \\ldots, 33$ are written respectively (one number per card). Members of the math club Vasya, Petya, and Misha decided to divide all these cards among themselves so that each of them gets at least one card and no one ends up with a pair of cards where the difference between the numbers is odd. How many ways are there to distribute the cards?",
        "answer": "4596",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 6062,
        "difficulty": null
      },
      {
        "id": "8f49f417e6c4ed883424d8b85010f1b659c201daeebefadbe971a0067141af30",
        "domain": "numina",
        "question": "Problem 10.4. Consider the sequence\n\n$$\na_{n}=\\cos (\\underbrace{100 \\ldots 0^{\\circ}}_{n-1})\n$$\n\nFor example, $a_{1}=\\cos 1^{\\circ}, a_{6}=\\cos 100000^{\\circ}$.\n\nHow many of the numbers $a_{1}, a_{2}, \\ldots, a_{100}$ are positive?",
        "answer": "99",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14029,
        "difficulty": null
      },
      {
        "id": "f586febd24758ebbb0f7840bf2db5b82bb212ebdd21b87e19572d0359ab620b9",
        "domain": "numina",
        "question": "4. In the store \"Everything for School,\" chalk is sold in packages of three grades: regular, unusual, and superior. Initially, the ratio of the quantities by grade was 3:4:6. As a result of sales and deliveries from the warehouse, this ratio changed to 2:5:8. It is known that the number of packages of superior chalk increased by $80\\%$, and the number of packages of regular chalk decreased by no more than 10 packages. How many packages of chalk were there in the store initially?\n\n(7 points)",
        "answer": "260",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8559,
        "difficulty": null
      },
      {
        "id": "73ae3a290a5b5898d4cd79f8643cb855e1cb87146c01cb9ca5f4048758f26a82",
        "domain": "numina",
        "question": "The Vaskapu rising above the city of Esztergom is seen from point $A$ near the railway station at an angle of $6^{\\circ} 50^{\\prime} 33^{\\prime \\prime}$, and from 200 meters closer at an angle of $7^{\\circ} 25^{\\prime} 52^{\\prime \\prime}$; from the same point $A$, the Sashegy is seen at an angle of $5^{\\circ} 24^{\\prime} 52^{\\prime \\prime}$, and from 200 meters closer at an angle of $5^{\\circ} 55^{\\prime} 36^{\\prime \\prime}$; the angle formed by the horizontal direction lines starting from point $A$ is $31^{\\circ} 4^{\\prime} 34^{\\prime \\prime}$. How far apart are the two mountain peaks from each other?",
        "answer": "1303",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 172508,
        "difficulty": null
      },
      {
        "id": "1251d7b2253439ae342ccdaf22f24e2eeebcff750c586c061dc93d4cdc24c138",
        "domain": "numina",
        "question": "G3.4 Let $x \\geq 0$ and $y \\geq 0$. Given that $x+y=18$. If the maximum value of $\\sqrt{x}+\\sqrt{y}$ is $d$, find the value of $d$. (Reference: 1999 FGS.2)",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152868,
        "difficulty": null
      },
      {
        "id": "378ca99cd6c8d11ebd1cad2987528e0de834b9510b27ff34c2b8af6f8defc21a",
        "domain": "numina",
        "question": "Draw two circles with the same center, one with a radius of $1 \\mathrm{~cm}$ and the other with a radius of $3 \\mathrm{~cm}$. In the region outside the circle with a radius of $1 \\mathrm{~cm}$ and inside the circle with a radius of $3 \\mathrm{~cm}$, draw circles that are simultaneously tangent to both circles, as shown in the figure below.\n\n(a) What should be the radius of these circles?\n\n(b) What is the maximum number of these circles, assuming they do not overlap?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_eead07372ad548e635bag-05.jpg?height=250&width=252&top_left_y=1214&top_left_x=1424)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73906,
        "difficulty": null
      },
      {
        "id": "f33a28fe8aaaa1406958351164ae911ae0ba37473bb5f328c09c0946bd96046f",
        "domain": "numina",
        "question": "## Problem 10.3. (Problem for the Atanas Radev award)\n\nFind all natural numbers $n$ such that: If $a$ and $b$ are natural numbers and $n$ divides $a^{2} b+1$, then $n$ divides $a^{2}+b$.",
        "answer": "240",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 177378,
        "difficulty": null
      },
      {
        "id": "bbd30c570537c4121adc9d9b50ba029f96a218f4f83c1a2d76dcebabd6bc93cd",
        "domain": "numina",
        "question": "13.429 First, the motorboat traveled $a$ km across the lake, and then half of this distance up the river flowing into the lake. The entire trip lasted 1 hour. Find the own speed of the motorboat if the speed of the river current is $c$ km/h.",
        "answer": "\\frac{3+2+\\sqrt{9^{2}-4+4^{2}}}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 52273,
        "difficulty": null
      },
      {
        "id": "adee1f867e8d2c7e5795ea0f817aeba5872b8a8f13ebd2ab9271ec3ca9f9cfe3",
        "domain": "numina",
        "question": "3. Find the sum\n$$\n\\frac{2^{1}}{4^{1}-1}+\\frac{2^{2}}{4^{2}-1}+\\frac{2^{4}}{4^{4}-1}+\\frac{2^{8}}{4^{8}-1}+\\cdots .\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152334,
        "difficulty": null
      },
      {
        "id": "8abfe687e993285bffc6fdad2bb683ed99d179b775423050871535ef53cb7989",
        "domain": "numina",
        "question": "2. (1992 Ukrainian Mathematical Olympiad) Find all prime numbers $p$ not exceeding 1000 such that $2p+1$ is a power of a natural number (i.e., there exist natural numbers $m$ and $n, n \\geqslant 2$, such that $2p+1=m^n$).",
        "answer": "13",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114048,
        "difficulty": null
      },
      {
        "id": "b10595020e301262692a11fe3aaa88b951be54f65de7a464a11885889b3c531a",
        "domain": "numina",
        "question": "1. Janez spent $\\frac{3}{5}$ of the money he had, then $\\frac{5}{9}$ of the remainder, and finally $\\frac{3}{8}$ of the new remainder. He was left with 80 euros. How much money did he have at the beginning?",
        "answer": "720",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163770,
        "difficulty": null
      },
      {
        "id": "59fa51d44b902867c801d66d411d9bef79c77db1a5f59d77104802190b32e16d",
        "domain": "numina",
        "question": "For positive numbers $a, b, c$ such that $a^{2}+b^{2}+c^{2}=1$. Determine the smallest possible value of\n\n$$\nS=\\frac{a b}{c}+\\frac{b c}{a}+\\frac{c a}{b}\n$$",
        "answer": "\\sqrt{3}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 171572,
        "difficulty": null
      },
      {
        "id": "5b6c5147502421554c3b6fe2fbc6a0e2b6acea2633dd9ba7064d6d9f64a9838c",
        "domain": "numina",
        "question": "# Task 2.\n\nWhile a lion cub, who was 6 minutes away from the water hole, set off to drink, a second one, having already quenched his thirst, started heading back along the same path at 1.5 times the speed of the first. At the same time, a turtle, who was 32 minutes away from the water hole, set off along the same path to drink. After some time, the first lion cub stepped on the turtle, and 2 minutes and 24 seconds later, the second lion cub stepped on her. How many minutes after the second incident did the turtle reach the water hole, given that all three moved at constant speeds?",
        "answer": "28.8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 6239,
        "difficulty": null
      },
      {
        "id": "ca534c098afe7591ec5050d74975f3e6c17250474cb791ce32228932c6175b90",
        "domain": "numina",
        "question": "3. (10 points) The distance between $A$ and $B$ is 500 kilometers. Two people, Jia and Yi, start cycling from $A$ to $B$ at the same time. Jia rides 30 kilometers per day, and Yi rides 50 kilometers per day but rests every other day. At the end of the $\\qquad$th day, the distance Yi has left to $B$ is twice the distance Jia has left to $B$.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134624,
        "difficulty": null
      },
      {
        "id": "6754da3ed2c7e729f64bfb98b731145c08c6e448d8fc466f7dd6bed30b402d72",
        "domain": "numina",
        "question": "Pista received the following task. \"Write the\n\n$$\n\\frac{16 e-10}{10 e-3}\n$$\n\nexpression, substituting integers for $e$, and simplify the resulting fraction if possible. Collect all the numbers with which you simplified!\"\n\nAfter several attempts, Pista found such a number and then proved that there are no more such numbers. Let's solve the problem too.",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 167421,
        "difficulty": null
      },
      {
        "id": "bbd071a1438414a23c56e93ad6ef2870b8df9cd18c392af9449e4e47a1f4eff2",
        "domain": "numina",
        "question": "11. (IMO-32 Shortlist) Find the largest positive integer $k$, such that\n$$\n1991^{k} \\mid 1990^{19911992}+1992^{19911990} \\text {. }\n$$",
        "answer": "1991",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 146061,
        "difficulty": null
      },
      {
        "id": "66e7268e22e449add2e4d9b4ad8d6b292416fe2eb7dbf67c51592962383347bc",
        "domain": "numina",
        "question": "Let $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ satisfy\n\n$f(1)=2, f(2)=1, f(3 n)=3 f(n), f(3 n+1)=3 f(n)+2, f(3 n+2)=3 f(n)+1$.\n\nFind how many integers $n \\leq 2014$ such that $f(n)=2 n$.",
        "answer": "128",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 83086,
        "difficulty": null
      },
      {
        "id": "ba7b3697a55ed98720dd78b9bf80d9bf85760076c4b7c97bf95c2c2e697959f4",
        "domain": "numina",
        "question": "During a school dance performance, a total of 430 different pairs danced. The first girl danced with 12 different boys, the second girl with 13 different boys, and so on, until the last girl danced with every boy. How many girls and how many boys attended the performance?",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 172778,
        "difficulty": null
      },
      {
        "id": "f57e13db60c95fe6eb781dad5aabfc349872f0b9de84c3d37a1aa1dd879dc1ac",
        "domain": "numina",
        "question": "4. With one mower, it is possible to mow $2 \\frac{1}{2}$ ha of meadow in 2 hours, and with another, smaller mower, $1 \\frac{1}{4}$ ha can be mowed in 1 hour and 30 minutes. How much meadow area will both mowers mow together in 3 hours and 36 minutes?",
        "answer": "7\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86991,
        "difficulty": null
      },
      {
        "id": "2c28b2582a88ee48497b33fdeb200941d8ea7cdb3c5ea41d09180cc3c2e978db",
        "domain": "numina",
        "question": "Example 22 (2002 National High School Mathematics Competition Question) Before the World Cup, Coach of Country $F$ is evaluating seven players, $A_{1}, A_{2}, \\cdots, A_{7}$, and plans to have them all play in three training matches (each 90 minutes long). Assuming that at any moment during the matches, exactly one of these players is on the field, and the total playing time (in minutes) of $A_{1}, A_{2}, A_{3}, A_{4}$ is divisible by 13. If there are no limits on the number of substitutions per match, how many different scenarios are there for the total playing time of each player?",
        "answer": "42244",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 130765,
        "difficulty": null
      },
      {
        "id": "a66c5f1fc111746310d97fc1cf9a24ba1e521085323c78065ac98953b8142f20",
        "domain": "numina",
        "question": "Ali-Baba came to a cave where there is gold, diamonds, and a chest in which he can carry them away. A full chest of gold weighs 200 kg, a full chest of diamonds - 40 kg, an empty chest weighs nothing. A kilogram of gold is worth 20 dinars on the market, a kilogram of diamonds - 60 dinars. Ali-Baba can lift and carry no more than 100 kg. What is the maximum amount of money he can get for the treasures he brings out of the cave in one go?",
        "answer": "3000",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 68029,
        "difficulty": null
      },
      {
        "id": "3fbefbbd2416af98ddfe223138a5208818b86a7545f6b764c7f25c8b34ae6cb1",
        "domain": "numina",
        "question": "Problem 3. How many odd three-digit numbers exist where the digit in the tens place is 4?",
        "answer": "45",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 188582,
        "difficulty": null
      },
      {
        "id": "5751726e707a8ec23c12704146297bcff623275b96a346246f0f5b045c69f1fb",
        "domain": "numina",
        "question": "10. As shown in the figure, it is a chessboard. The chess piece \"Knight\" can move from one square, jumping 2 squares horizontally or vertically, and at the same time jumping 1 square in the other direction, forming an \"L\" shape. For example, from point A in the figure, it can reach any of the B positions in one step. Then, for a chess piece \"Knight\" to jump from the bottom-left corner of the chessboard to the top-right corner, the minimum number of steps required is $\\qquad$.",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 111482,
        "difficulty": null
      },
      {
        "id": "b25d2d692908591330e3e138b3b4ed4da8c44d79cf0907156d2bb0909161fb01",
        "domain": "numina",
        "question": "9. Consider a triangle $A B C$ and a point $D$ on the side $A C$. If $\\overline{A B}=\\overline{D C}=1, \\angle D B C=30^{\\circ}$ and $\\angle A B D=90^{\\circ}$, calculate the value of $\\overline{A D}$.",
        "answer": "\\sqrt[3]{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86236,
        "difficulty": null
      },
      {
        "id": "4d65dc432c4ca1237161c5ce223c6aaee0552fb6ba331911814c97d9da13ac46",
        "domain": "numina",
        "question": "15. Each letter in the figure represents a number, and the sum of the three numbers in each circle is 21, and the sum of all the numbers is 69. Then $b=$ $\\qquad$ .",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 123282,
        "difficulty": null
      },
      {
        "id": "54a068848cbbd6d14967f5f4370fb8f2e22d2b79e1c6f2f9b9c63dad1f1893e4",
        "domain": "numina",
        "question": "Problem 7.8. Carlson and Little Man have several jars of jam, each weighing an integer number of pounds.\n\nThe total weight of all the jars of jam that Carlson has is 13 times the total weight of all the jars that Little Man has. Carlson gave Little Man the jar with the smallest weight (from those he had), after which the total weight of his jars turned out to be 8 times the total weight of Little Man's jars.\n\nWhat is the maximum number of jars of jam that Carlson could have had initially?",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12888,
        "difficulty": null
      },
      {
        "id": "cf21848d9050b314a2d48ea36dc4333e6ea3171de7e61da5a9c55a2cb0143c89",
        "domain": "numina",
        "question": "2. Let $a$ and $b$ be real numbers such that $a>b>0$ and $a^{2}+b^{2}=6ab$. Determine the value of the expression $\\frac{a+b}{a-b}$.",
        "answer": "\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189116,
        "difficulty": null
      },
      {
        "id": "b86f933582b15c50d32ce2296f11abae960b6ad23ae7689cfd99fbf24e47c451",
        "domain": "numina",
        "question": "5. First-grader (6th grade. 2 points). Maksim will go to the first grade on September 1, 2014, at the age of 6 years, and this day will not be his birthday. What is the probability that he was born in 2008?",
        "answer": "\\frac{244}{365}",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 10898,
        "difficulty": null
      },
      {
        "id": "2808c0cb48986695f2c74fe6ef7e4437aaef6b8a56485fd29e4e7f51e9c3d507",
        "domain": "numina",
        "question": "37. In two boxes, there are 70 coins. It is known that in the first box, $\\frac{5}{9}$ of the total number of coins are gold, and the rest are silver, in the second box, $\\frac{7}{17}$ of the number of coins are silver, and the rest are gold. How many coins are in each box",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35795,
        "difficulty": null
      },
      {
        "id": "14b679ab09176f3c43b43a6e6d6ee60aaf74f7ef10a14bdd1db6749c5187158e",
        "domain": "numina",
        "question": "4・183 Find the number of real number pairs $(a, b)$ that make the system of equations $\\left\\{\\begin{array}{l}a x+b y=1, \\\\ x^{2}+y^{2}=50 .\\end{array}\\right.$ have at least one solution, and all solutions are integer solutions.",
        "answer": "72",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119922,
        "difficulty": null
      },
      {
        "id": "3323ceeabc1ebc5dd89601e94e54d88fb70e46b72a5829d827be39859454665b",
        "domain": "numina",
        "question": "4. Among the integers from 1 to 2020, there are 1616 numbers that are not multiples of 5. Divide these 1616 numbers into several groups (the numbers in each group do not have to be the same), such that the difference (larger number minus smaller number) between any two numbers in each group is a prime number. What is the minimum number of groups needed?",
        "answer": "404",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113759,
        "difficulty": null
      },
      {
        "id": "18cfffa6a2d30518d67c0e298abcbdf3c66f491c92e8b904eedcc2c7cde6bdad",
        "domain": "numina",
        "question": "3. Calculate: $\\frac{1}{1+\\frac{2}{1+\\frac{3}{1+\\frac{4}{5}+1}+1}+1}$.",
        "answer": "\\frac{43}{114}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131048,
        "difficulty": null
      },
      {
        "id": "b6773cea50f82e757649ca149d7b2acce5f4d8e5ff8cd3980940ca5643ab3575",
        "domain": "numina",
        "question": "8. (15 points) A primary school is conducting a height statistics survey. There are 99 students whose height does not exceed $130 \\mathrm{~cm}$, with an average height of $122 \\mathrm{~cm}$. There are 72 students whose height is not less than $160 \\mathrm{~cm}$, with an average height of $163 \\mathrm{~cm}$. The average height of students whose height exceeds $130 \\mathrm{~cm}$ is $155 \\mathrm{~cm}$, and the average height of students whose height is below $160 \\mathrm{~cm}$ is $148 \\mathrm{~cm}$. How many students are there in total? $\\qquad$",
        "answer": "621",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 126092,
        "difficulty": null
      },
      {
        "id": "d4358b8fc8a9da16509699b8d724b4872fc3186818c3be86d4dca4fab849f103",
        "domain": "numina",
        "question": "3.040. $\\frac{1-\\cos 4 \\alpha}{\\cos ^{-2} 2 \\alpha-1}+\\frac{1+\\cos 4 \\alpha}{\\sin ^{-2} 2 \\alpha-1}=2$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50532,
        "difficulty": null
      },
      {
        "id": "d3f0192f6cc753d57bef17571faa6c722ec1f36d477679f98c8bad6795b5d4b6",
        "domain": "numina",
        "question": "(Example 3 Given that the function $y=f(x)$ has an inverse function within its domain $(-\\infty, 0]$, and $f(x-1)=x^{2}-2 x$, find the value of $f^{-1}\\left(-\\frac{1}{2}\\right)$.",
        "answer": "-\\frac{\\sqrt{2}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95456,
        "difficulty": null
      },
      {
        "id": "27f63cb0a678b16cb2d34d8b932d9868b20b8e86ffa469a932d6948e56c8137d",
        "domain": "numina",
        "question": "Example 5. Expand the function\n\n$$\nf(z)=\\frac{z}{z^{2}-2 z-3}\n$$\n\ninto a Taylor series in the neighborhood of the point $z_{0}=0$ using expansion (12), and find the radius of convergence of the series.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 31048,
        "difficulty": null
      },
      {
        "id": "9e5785ec69d454d95d3c67acaa2989d83d455da503793dbf4b21a69012b0c902",
        "domain": "numina",
        "question": "## Task B-2.1.\n\nFor which $n \\in \\mathbb{N}$ does the following hold\n\n$$\n1 i^{1}+2 i^{2}+3 i^{3}+4 i^{4}+\\cdots+n i^{n}=48+49 i, \\text { if } i=\\sqrt{-1} ?\n$$",
        "answer": "97",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88617,
        "difficulty": null
      },
      {
        "id": "5c1528e2049b1339b2ac4ff926765527b952e21218305af521d9940180b59c18",
        "domain": "numina",
        "question": "## 7. Gold Coins\n\nAnte, Branimir, Celestin, and Dubravko are sitting around a round table (in that order). Together, they have 1600 gold coins. First, Ante divides half of his gold coins into two equal parts and gives one part to his left neighbor and the other part to his right neighbor, while keeping the other half for himself. Then, Branimir does the same, followed by Celestin, and finally Dubravko. In the end, all four of them have an equal number of gold coins. How many gold coins did Branimir have at the beginning?\n\nResult: $\\quad 575$",
        "answer": "575",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 89880,
        "difficulty": null
      },
      {
        "id": "b098d49bb9555a56324301aea05fd5a76b055111b2662c1f5140fcdd65817487",
        "domain": "numina",
        "question": "5. In $\\triangle A B C$, $\\angle B A C=90^{\\circ}, A C=A B=4$, point $D$ is inside $\\triangle A B C$, and $A D=\\sqrt{2}$. Then the minimum value of $B D+C D$ is $\\qquad$ .",
        "answer": "2\\sqrt{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 96644,
        "difficulty": null
      },
      {
        "id": "d04f84c8465b80e3b2e15276fd09eeb850b05085f7f16aadcb3c53847d08d96a",
        "domain": "numina",
        "question": "Given an equilateral triangle $ABC$ with side length 4. Points $D$, $E$, and $F$ are on $BC$, $CA$, and $AB$ respectively, and $|AE| = |BF| = |CD| = 1$. Connect $AD$, $BE$, and $CF$, intersecting to form $\\triangle RQS$. Point $P$ moves within $\\triangle RQS$ and on its sides. The distances from $P$ to the three sides of $\\triangle ABC$ are denoted as $x$, $y$, and $z$ respectively.\n(1) Prove that when point $P$ is at the vertices of $\\triangle RQS$, the product $xyz$ has a minimum value; $\\square$\n(2) Find the minimum value of the product $xyz$.",
        "answer": "\\frac{648}{2197}\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 109654,
        "difficulty": null
      },
      {
        "id": "13722ee565f0cfeae42d99078601b8f40929cb7fd420291ec7d54ebc077ccb94",
        "domain": "numina",
        "question": "7.1. A cyclist was initially traveling at a speed of 20 (km/h). After covering one-third of the distance, he glanced at his watch and decided to increase his speed by 20%. He continued the rest of the journey at the new speed. What was the cyclist's average speed?",
        "answer": "22.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 22533,
        "difficulty": null
      },
      {
        "id": "1b8ac26a7166372bed9b963328b6702471bdad5799066b1e37e2f5d29ba92821",
        "domain": "numina",
        "question": "Example 2 Use $1 \\times 1, 2 \\times 2, 3 \\times 3$ tiles to cover a $23 \\times 23$ floor (no overlapping, no gaps), how many $1 \\times 1$ tiles are needed at least?",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 138642,
        "difficulty": null
      },
      {
        "id": "7bb7b6075c150ebf1e842baa29ba7f535f08e22e74f1b58a2f7229ddef29c68b",
        "domain": "numina",
        "question": "12. The ordered pairs $(2011,2),(2010,3),(2009,4), \\ldots,(1008,1005),(1007,1006)$ are written from left to right on a blackboard. Every minute, Elizabeth selects a pair of adjacent pairs $\\left(x_{i}, y_{i}\\right)$ and $\\left(x_{j}, y_{j}\\right)$, with $\\left(x_{i}, y_{i}\\right)$ left of $\\left(x_{j}, y_{j}\\right)$, erases them, and writes $\\left(\\frac{x_{i} y_{i} x_{j}}{y_{j}}, \\frac{x_{i} y_{i} y_{j}}{x_{j}}\\right)$ in their place. Elizabeth continues this process until only one ordered pair remains. How many possible ordered pairs $(x, y)$ could appear on the blackboard after the process has come to a conclusion?",
        "answer": "504510",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151319,
        "difficulty": null
      },
      {
        "id": "99258abec88324c9a5d9506d5fe5d66d8158a2e05f07e8e339e1413281d44a3c",
        "domain": "numina",
        "question": "7.2. Find the GCD of all numbers obtained by all possible permutations of the digits of the number 202120222023\n\nSolution. By the divisibility rule, all these numbers are divisible by 9 (the sum of the digits is 18). A sufficient condition to prove that there are no other numbers is that the difference between any two such numbers is also divisible by the GCD. For example, 222222100032 - 222222100023 = 9 is divisible by 9 and cannot be greater than 9.",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17873,
        "difficulty": null
      },
      {
        "id": "597d34aab32c416dd96e84d7c609bb62157c9352410a50af19e4b5de696f68da",
        "domain": "numina",
        "question": "## Task 4.\n\nDetermine all natural numbers $n$ for which there exist distinct divisors $a$ and $b$ of $n$ such that there are no other divisors of $n$ between them and that\n\n$$\nn=a^{2}-b\n$$",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 89851,
        "difficulty": null
      },
      {
        "id": "6bcf68dfd18d61ddadeb21b06f1e4eead674855d4d931f875bd336605faf3c23",
        "domain": "numina",
        "question": "3 [ Volume of a Tetrahedron and Pyramid $\\quad$]\n\nConsider a rectangle $A B C D$, where $A B=2, B C=3$. Segment $K M$ is parallel to $A B$ (see figure), located 1 unit away from the plane $A B C D$ and $K M=5$. Find the volume of the polyhedron $A B C D K M$.\n\n#",
        "answer": "\\frac{9}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66481,
        "difficulty": null
      },
      {
        "id": "0afa9722914a61d1a0410c62f67c87d17df00667d42e7dc1fd4760a2e742490d",
        "domain": "numina",
        "question": "Example 4 Find the largest constant $c$ such that for all real numbers $x, y$ satisfying $x>0, y>0, x^{2}+y^{2}=1$, we always have $x^{6}+y^{6} \\geqslant c x y$.",
        "answer": "\\frac{1}{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 126595,
        "difficulty": null
      },
      {
        "id": "e1ffb4ff90a458246c18ba6e9512022fdfb96b832f648b1fffe310d00d736898",
        "domain": "numina",
        "question": "5. (3 points) In a regular triangular pyramid (not a regular tetrahedron), the area of the base is half the area of a lateral face. The height of the pyramid is 70 cm.\n\nConsider the following (infinite) sequence of spheres. Let \\( S_{1} \\) be the inscribed sphere of this pyramid. Then \\( S_{2} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{1} \\); \\( S_{3} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{2} \\), not equal to \\( S_{1} \\), and so on; \\( S_{n+1} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{n} \\), not equal to \\( S_{n-1} \\).\n\nFind the total volume of all these spheres.",
        "answer": "\\frac{686}{327}\\pi^3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20628,
        "difficulty": null
      },
      {
        "id": "3bb091b586e28bbdf0a1540c1b2023e315959fbaba8781102a107db0fd1b9fe6",
        "domain": "numina",
        "question": "Beatriz adores odd numbers. How many numbers between 0 and 1000 can she write using only odd digits?",
        "answer": "155",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74433,
        "difficulty": null
      },
      {
        "id": "d43ffeff46e3ac1822ce4b51e73366049f4be40f0b67417d2f91cb2b0941561b",
        "domain": "numina",
        "question": "1. (8 points) The calculation result of the expression $2016 \\times\\left(\\frac{8}{7 \\times 9}-\\frac{1}{8}\\right)$ is",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122324,
        "difficulty": null
      },
      {
        "id": "9181f1d252523a60d144b93c53269644003791d0b4e7f0ca47307a012a9dc21e",
        "domain": "numina",
        "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhich number is greater: $100^{100}$ or $50^{50} \\cdot 150^{50}$ ?",
        "answer": "100^{100}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 69032,
        "difficulty": null
      },
      {
        "id": "6648ee233c9a75719d8965eb046ca39a9b9110d438c37126a2f03528086f3def",
        "domain": "numina",
        "question": "Example 3.28. Compute the double integral $\\iint_{D}(x+2 y) d x d y$, where the region of integration is bounded by the parabolas $y=x-x^{2}$, $y=1-x^{2}$, and the $O y$ axis (Fig. 48).\n\n![](https://cdn.mathpix.com/cropped/2024_05_22_e915d089cc29fd681be5g-177.jpg?height=450&width=534&top_left_y=175&top_left_x=441)\n\nFig. 48",
        "answer": "\\frac{2}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30325,
        "difficulty": null
      },
      {
        "id": "8f4aa1c8e88e0304bc7bc9c4d35ff00396737585b9713b5d65c311d4ed80ade9",
        "domain": "numina",
        "question": "3.2. A chocolate bar weighed 400 g and cost 150 rubles. Recently, to save money, the manufacturer reduced the weight of the bar to 300 g, and increased its price to 180 rubles. By what percentage did the manufacturer's revenue increase?",
        "answer": "60",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16524,
        "difficulty": null
      },
      {
        "id": "c546967c7aaff4950fc197f23386df25055a5f0692518775c2e7c937fea60b08",
        "domain": "numina",
        "question": "Problem 6. Calculate $2 \\operatorname{arctg} 5+\\arcsin \\frac{5}{13}$.",
        "answer": "\\pi",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9456,
        "difficulty": null
      },
      {
        "id": "b5913eda0fa988551c55a54baa3f009187ce8e288d51f079822ee034035cb4ee",
        "domain": "numina",
        "question": "A $20 \\times 20 \\times 20$ cube is divided into 8000 unit cubes. A number is written in each unit cube. In each row and in each column of 20 small cubes, parallel to one of the cube's edges, the sum of the numbers is 1. In one of the small cubes, the number written is 10. Through this small cube pass three layers $1 \\times 20 \\times 20$ parallel to the faces of the cube. Find the sum of all the numbers outside these three layers.\n\n## - Solution of the exercises -\n\n untranslated portion: \n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。\n\n---\n\nA $20 \\times 20 \\times 20$ cube is divided into 8000 unit cubes. A number is written in each unit cube. In each row and in each column of 20 small cubes, parallel to one of the cube's edges, the sum of the numbers is 1. In one of the small cubes, the number written is 10. Through this small cube pass three layers $1 \\times 20 \\times 20$ parallel to the faces of the cube. Find the sum of all the numbers outside these three layers.\n\n## - Solution of the exercises -",
        "answer": "333",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 83534,
        "difficulty": null
      },
      {
        "id": "89c91b52f0aa79da1bd086bdc67de7363a8604f41c862e0307936adc614601ee",
        "domain": "numina",
        "question": "11. For a positive integer $n$, select several positive divisors of $n$ such that these divisors do not have an integer multiple relationship with each other. Let the maximum number of divisors that can be selected be $f(n)$, then $f\\left(2^{2020} \\times 2020\\right)+f\\left(3^{2021} \\times 2021\\right)+f\\left(5^{2022} \\times 2022\\right)=$ $\\qquad$ .",
        "answer": "16",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 106003,
        "difficulty": null
      },
      {
        "id": "c46f41f28fdc8c3ef8b3de10b76e18f60102fa48a4132208d30021d03bf9960d",
        "domain": "numina",
        "question": "2. Find all natural numbers $n$ for which the value of the expression $\\left[\\frac{n^{2}}{4}\\right]$ is a square of an integer.\n\nNOTE. $\\mathrm{S}[x]$ denotes the integer part of the number $x$, which is the greatest integer less than or equal to $x$.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162894,
        "difficulty": null
      },
      {
        "id": "9009ab0866be39eff21b146cfc8c94c3902a1cd6b152cba1d26e2743e79470ee",
        "domain": "numina",
        "question": "$4 \\cdot 34$ An integer has exactly 4 prime factors, the sum of whose squares is 476. Find this integer.\n(China National Training Team Practice Question, 1990)",
        "answer": "1989",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114431,
        "difficulty": null
      },
      {
        "id": "5544f9731a9a5d23fd3526552312a26ce0a3a6e7369b9480f90abdacaa437916",
        "domain": "numina",
        "question": "[ Arithmetic. Mental calculation, etc.]\n\nOn the mailbox, it is written: \"Mail is collected five times a day from 7 to 19 hours.\" Indeed, the first time the postman collects the mail is at 7 in the morning, and the last time is at 7 in the evening. At what intervals are the letters taken out of the box?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 69088,
        "difficulty": null
      },
      {
        "id": "3aa309475aa6e4d5508cea984a2c9f786429ebb55ffcc7e7273d6e0c2e5450c2",
        "domain": "numina",
        "question": "131. A natural number $n$ is the product of four consecutive natural numbers, each greater than 5. Determine the maximum number of possible last digits of $n$, given that its last digit is not 0.",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 36055,
        "difficulty": null
      },
      {
        "id": "9a7076bea50d1d7e211da0a344525472649ca414e91ca580a06af2e73ae88d7b",
        "domain": "numina",
        "question": "1. (8 points) Calculate: $6 \\times\\left(\\frac{1}{2}-\\frac{1}{3}\\right)+12 \\times\\left(\\frac{1}{3}+\\frac{1}{4}\\right)+19-33+21-7+22=$ $\\qquad$",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92187,
        "difficulty": null
      },
      {
        "id": "b732e896c1fbe0e1b24c417e3ed5590cd0f92fa80f573f6b8a57b8ea7b9be7cc",
        "domain": "numina",
        "question": "I2.2 In the trapezium $A B C D, A B / / D C . A C$ and $B D$ intersect at $O$. The areas of triangles $A O B$ and $C O D$ are $P$ and 25 respectively. Given that the area of the trapezium is $Q$, find the value of $Q$.",
        "answer": "81",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152933,
        "difficulty": null
      },
      {
        "id": "6cfdf444d8b447d457049f0b476ad6c22ffc361656386318573a3cb4388d95fb",
        "domain": "numina",
        "question": "## Task B-4.6.\n\nIn an isosceles triangle with a base length of $14 \\mathrm{~cm}$ and legs of length $25 \\mathrm{~cm}$, a circle is inscribed. Above it, another circle is inscribed which touches both legs and the first circle, above this one a third circle is inscribed, and so on, continuing the process of inscribing circles. If all the inscribed circles are painted with a certain color, what part of the triangle's area will remain uncolored?",
        "answer": "168-\\frac{1008}{25}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88635,
        "difficulty": null
      },
      {
        "id": "c515878b0c2d2b01dcd755ef28324659404290d26bfba629c9767b317fa88e8c",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\tan\\left(\\frac{\\pi}{4}-x\\right)\\right)^{\\left(e^{x}-1\\right) / x}$",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46980,
        "difficulty": null
      },
      {
        "id": "635cab7ed74075522eb37f71755967f47c547fb665608ca7e60a992c91e5dfbe",
        "domain": "numina",
        "question": "76(1010). A collective farm team must harvest a potato crop by a certain deadline. After 60% of the potatoes were harvested, a combine was sent to assist the team, which reduced the harvesting time by 5 days. How many days would it have taken to harvest the potatoes without the combine's help, if it is known that the combine could have completed the entire job 8 days faster than the team?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36010,
        "difficulty": null
      },
      {
        "id": "3f5fbb5fbd7a70521b9b754ea558d706e10bfdb412058cc6fc84a41fc9d9b216",
        "domain": "numina",
        "question": "20. Xiaoling, while calculating division, wrote the divisor 65 as 56, resulting in a quotient of 13 and a remainder of 52. The correct quotient should be $\\qquad$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134668,
        "difficulty": null
      },
      {
        "id": "f76e4ae384a8b44b7b2ae289902d39013ebdf62f7a4ce3fdcb304bda0c05d2a6",
        "domain": "numina",
        "question": "2. A clock takes 12 seconds to strike 7 times, how many seconds does it take to strike 4 times?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 138591,
        "difficulty": null
      },
      {
        "id": "ad03d790586079a6f254f226d440227de2d301c85095072e3f6e4474b0d1eac7",
        "domain": "numina",
        "question": "84. As shown in the figure, Jia and Yi are two cylindrical containers, with the ratio of their base areas being $2: 3$. In the Jia container, there is an iron ball with a volume of 30 cubic centimeters, at which point the water levels in the two containers differ by $x$ centimeters; if the iron ball is moved from the Jia container to the Yi container, the water levels in the two containers still differ by $x$ centimeters. If the base area of the Jia container is 25 square centimeters, $x=$ $\\qquad$ cm.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126702,
        "difficulty": null
      },
      {
        "id": "a75f1e298cbdacc752822973e6e58c9f0954e925b9f04284411374351f50b995",
        "domain": "numina",
        "question": "Three. (This question is worth 50 points) Given 10 points in space, where no four points lie on the same plane. Connect some of the points with line segments. If the resulting figure contains no triangles and no spatial quadrilaterals, determine the maximum number of line segments that can be drawn.\n\nConnect some of the points with line segments, if the resulting figure contains no triangles and no spatial quadrilaterals, determine the maximum number of line segments that can be drawn.",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 134778,
        "difficulty": null
      },
      {
        "id": "b43c939c0f82c6e0737ffd8987cc353fb4d78060ae9b840b011d98b6b217ce6f",
        "domain": "numina",
        "question": "Task B-4.3. Find all solutions to the equation\n\n$$\n\\frac{1}{\\binom{4}{x}}-\\frac{1}{\\binom{5}{x}}=\\frac{1}{\\binom{6}{x}}\n$$",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 87934,
        "difficulty": null
      },
      {
        "id": "59521a09b4083949beafcf1e902ca3fcf23953dbf85127f51bfbaf954fff9893",
        "domain": "numina",
        "question": "Example 6 Find all prime numbers $p$ such that $4 p^{2}+1$ and $6 p^{2}+1$ are also prime.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 134453,
        "difficulty": null
      },
      {
        "id": "0983aeff7e4e763f23818012cbcabc23897912696fae6c1ee77bbd82b2160d9f",
        "domain": "numina",
        "question": "5. Solve the equation $\\frac{\\sqrt{(-x)^{2}}+(\\sqrt{-x})^{2}}{x^{2}+(-x)^{2}}=\\frac{1}{2016}$ ANSWER: $x=-2016$.",
        "answer": "-2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5481,
        "difficulty": null
      },
      {
        "id": "b374ccb3bde1b13998888844f8c197c02330672a0346e996b2d1fcf9d346d707",
        "domain": "numina",
        "question": "6.2. On the plate, there were 15 doughnuts. Karlson took three times more doughnuts than Little Man, and Little Man's dog Bibbo took three times fewer than Little Man. How many doughnuts are left on the plate? Explain your answer.",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 15514,
        "difficulty": null
      },
      {
        "id": "92416f9b607c1382f1906663dd3b8ce5e23523504b19da8c3f6deee60930e6f9",
        "domain": "numina",
        "question": "11. Let $a \\in \\mathbf{R}$, the equation ||$x-a|-a|=2$ has exactly three distinct roots, then $a=$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108290,
        "difficulty": null
      },
      {
        "id": "2c07a80f6c888ff48110c41f663fb9724ebbcb3dfc9c6e3f9f21ad6872ba825e",
        "domain": "numina",
        "question": "20. As shown in the figure, with the three sides of the isosceles right triangle $A B C$ as sides, squares are constructed outward. Given that the area of triangle $A B C$ is 4, then the area of the hexagon $E F G H I J$ formed by the vertices of the squares is $\\qquad$",
        "answer": "48",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 136372,
        "difficulty": null
      },
      {
        "id": "4befd2c5e34ffab260ab0524aeaeb393f41c1ecee7f5c9830126d2624a74b459",
        "domain": "numina",
        "question": "11. In the arithmetic expression below, each square represents a digit. Question: What is the sum of the 6 digits in the squares?\n$$\n\\begin{array}{r}\n\\square \\square \\square \\\\\n+\\square \\square \\square \\\\\n\\hline 1991\n\\end{array}\n$$",
        "answer": "47",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 124236,
        "difficulty": null
      },
      {
        "id": "8db96ae86dc1fa4c651a216e1f4e9c713b5a10b5e42117e052e2cf11849cfc3f",
        "domain": "numina",
        "question": "[ Ratio of areas of similar triangles ] Law of Sines\n\nIn an acute-angled triangle $A B C$, heights $A P$ and $C Q$ are dropped from vertices $A$ and $C$ to sides $B C$ and $A B$. It is known that the area of triangle $A B C$ is 18, the area of triangle $B P Q$ is 2, and $P Q=2 \\sqrt{2}$. Find the radius of the circumcircle of triangle $A B C$.",
        "answer": "\\frac{9}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66153,
        "difficulty": null
      },
      {
        "id": "d1131bf645f7cc446e6c112877f3a3c107f3dc4dce7d98018777063b979e11ba",
        "domain": "numina",
        "question": "Booin d.A.\n\nGiven two sequences: $2,4,8,16,14,10,2$ and 3, 6, 12. In each of them, each number is obtained from the previous one according to the same rule.\n\na) Find this rule.\n\nb) Find all natural numbers that transform into themselves (according to this rule).\n\nc) Prove that the number $2^{1991}$ will become a single-digit number after several transformations.",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 54109,
        "difficulty": null
      },
      {
        "id": "78ebdfacbb10207a92c35b17b68ddb6afd102038ed4b1970d14a96053d09ffb9",
        "domain": "numina",
        "question": "G1.3 Determine the remainder of the 81 -digit integer $\\overline{111 \\cdots 1}$ divided by 81 .",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153362,
        "difficulty": null
      },
      {
        "id": "f785d41d87f7b1abe04d9f9efd0e763ae153e183bd1259627a35bd9f0b2bcbb4",
        "domain": "numina",
        "question": "14. A, B, C, and D are competing in a table tennis tournament, where every two individuals play one match against each other. As a result, A defeated D, and A, B, and C have the same number of wins. How many matches did D win?",
        "answer": "0",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 98374,
        "difficulty": null
      },
      {
        "id": "39a6941579dbee97a367769befb3b63862305d8b2e388ced74c7b5bd1afa564a",
        "domain": "numina",
        "question": "Task 2. Angelica wants to choose a three-digit code for her suitcase. To make it easier to remember, Angelica wants all the digits in her code to be in non-decreasing order. How many different options does Angelica have for choosing the code?",
        "answer": "220",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9195,
        "difficulty": null
      },
      {
        "id": "202743d2027ab90d3a958db631e75da9a78379e9f4a24d63ac82c1d571468823",
        "domain": "numina",
        "question": "Task 1. Represent in the form of an irreducible fraction\n\n$$\n6 \\frac{7}{2015} \\times 4 \\frac{5}{2016}-7 \\frac{2008}{2015} \\times 2 \\frac{2011}{2016}-7 \\times \\frac{7}{2015}\n$$",
        "answer": "\\frac{5}{144}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9451,
        "difficulty": null
      },
      {
        "id": "39e5f4146ca581d04f7db8c203831e6c6fadcf245b60e270123de80b66202703",
        "domain": "numina",
        "question": "Exercise 3. Let $x, y$ and $z$ be three real numbers such that $x^{2}+y^{2}+z^{2}=1$. Find the minimum and maximum possible values of the real number $x y+y z-z x$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 77431,
        "difficulty": null
      },
      {
        "id": "85c45272c04fd273dce6c315b92ef4f62b24b64feee76d602268bc3e1da50e9d",
        "domain": "numina",
        "question": "2k* Given that the base of the tetrahedron $S-ABC$ is an equilateral triangle, the projection of point $A$ on the side face $SBC$ is $H$, which is the orthocenter of $\\triangle SBC$, the plane angle of the dihedral angle $H-AB-C$ is $30^{\\circ}$, and $SA=2\\sqrt{3}$. Then, the volume of the tetrahedron $S-ABC$ is $\\qquad$ .",
        "answer": "\\frac{9}{4}\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133782,
        "difficulty": null
      },
      {
        "id": "a811fcc283172e2f21b7ef0c89e20f7d6c7a45df1381223290ae522fa885b0a4",
        "domain": "numina",
        "question": "8. Priti is learning a new language called Tedio. During her one hour lesson, which started at midday, she looks at the clock and notices that the hour hand and the minute hand make exactly the same angle with the vertical, as shown in the diagram. How many whole seconds remain until the end of the lesson?",
        "answer": "276",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156456,
        "difficulty": null
      },
      {
        "id": "7116b2b898e8132830c9a3cf754f7c4cd598850138b23783fbec95287098906e",
        "domain": "numina",
        "question": "4. In the regular tetrahedron \\(ABCD\\), consider point \\(M\\) - the midpoint of \\([AB]\\) and \\(N\\) - the midpoint of \\([AC]\\). If the edge of the tetrahedron is \\(8 \\, \\text{cm}\\), determine:\n\n(3 p) a) The perimeter of quadrilateral \\(BCNM\\)\n\n(2 p) b) The measure of the angle formed by line \\(MN\\) with \\(AB\\)\n\n(2 p) c) The distance from point \\(C\\) to the plane \\((ABD)\\).\n\nNote: All subtasks are mandatory,\n\nTime allowed: 3 hours\n\nEach subtask is graded from 0 to 7 points.\n\n## GRADING AND SCORING CRITERIA FOR THE MATHEMATICS OLYMPIAD LOCAL PHASE",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 178556,
        "difficulty": null
      },
      {
        "id": "8f9c1ba1a235a495b538c4c71f62da91ff26ec7af8976bbb8b3f87f1bcf8bd71",
        "domain": "numina",
        "question": "## Task 5 - 040935\n\nAt a puzzle afternoon, the best young mathematician in the class is given the task of guessing a certain real number. His classmates take turns naming properties of this number:\n\nKlaus: \"The number is divisible by 4 without a remainder.\"\n\nInge: \"The number is the radius of a circle whose circumference is 2.\"\n\nGünter: \"The number is less than 3.\"\n\nMonika: \"The number is the length of the diagonal of a square whose side has a length of 2.\"\n\nBärbel: \"The number is irrational.\"\n\nPeter: \"The number is the area of an equilateral triangle whose side has a length of 2.\" Furthermore, he learns that of the students Klaus and Inge, Günter and Monika, and Bärbel and Peter, exactly one of each pair spoke the truth.\n\nWhat is the number?",
        "answer": "\\frac{1}{\\pi}",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 184414,
        "difficulty": null
      },
      {
        "id": "6127776bf3bc336bd0c7d2c81a81749b5297dab70156ec01693073d5a4246fd0",
        "domain": "numina",
        "question": "Example 3 Let the complex numbers $z_{1}, z_{2}, z_{3}$ satisfy:\n$$\n\\begin{array}{l}\n\\left|z_{1}\\right|=\\left|z_{2}\\right|=\\sqrt{3}\\left|z_{3}\\right|, z_{1}+z_{3}=z_{2} . \\\\\n\\text { Find the value of } \\frac{z_{1} z_{2}}{z_{3}} \\text { . }\n\\end{array}\n$$",
        "answer": "-5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127604,
        "difficulty": null
      },
      {
        "id": "f9652029b6e443f0c69414df743d34acb48c9ce50f1d7dd96a1fd7e8c92029f8",
        "domain": "numina",
        "question": "A fly and $\\mathrm{k}$ spiders move on a $2019 \\times 2019$ grid. On their turn, the fly can move 1 square and the $k$ spiders can each move 1 square. What is the minimal $k$ for which the spiders are sure to catch the fly.",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 79370,
        "difficulty": null
      },
      {
        "id": "bf73e27608769b121e6eb11b80a67469dd831f73a55d02e71f7565d84009dc25",
        "domain": "numina",
        "question": "Example 2 The equation $x^{10}+(13 x-1)^{10}=0$ has 10 complex roots $r_{1}, \\overline{r_{1}}, \\overline{r_{2}}, \\overline{r_{2}}, \\overline{r_{3}}, \\overline{r_{3}}, \\overline{r_{4}}, \\overline{r_{4}}, \\overline{r_{5}}, \\overline{r_{5}}$, where $\\overline{r_{i}}$ is the conjugate of $r_{i}$ $(i=1,2,3,4,5)$, find the value of $\\frac{1}{r_{1} \\overline{r_{1}}}+\\frac{1}{r_{2} \\overline{r_{2}}}+\\frac{1}{r_{3}} \\overline{r_{3}}+\\frac{1}{r_{4} \\overline{r_{4}}}+\\frac{1}{r_{5} r_{5}}$.",
        "answer": "850",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143999,
        "difficulty": null
      },
      {
        "id": "9cd97f40eb3139ab9f656f08c8bfa685dc7a219e80583ca8fd5307a01ab88275",
        "domain": "numina",
        "question": "2. [5 points] Consider all possible tetrahedra $ABCD$ in which $AB=2$, $AC=CB=7$, and $AD=DB=8$. Each such tetrahedron is inscribed in a cylinder such that all vertices lie on its lateral surface, and the edge $CD$ is parallel to the axis of the cylinder. Choose the tetrahedron for which the radius of the cylinder is the smallest among those obtained. What values can the length of $CD$ take in such a tetrahedron?",
        "answer": "\\sqrt{62}\\\\sqrt{47}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 3367,
        "difficulty": null
      },
      {
        "id": "06ec2dc17cebc3aee776a453c8f3b18d9fc820e01b5b4626129030a573ef4b29",
        "domain": "numina",
        "question": "Problem 9.4. A line $\\ell$ is drawn through vertex $A$ of rectangle $ABCD$, as shown in the figure. Perpendiculars $BX$ and $DY$ are dropped from points $B$ and $D$ to line $\\ell$. Find the length of segment $XY$, given that $BX=4$, $DY=10$, and $BC=2AB$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_3890a5f9667fd1ab5160g-30.jpg?height=426&width=515&top_left_y=1495&top_left_x=469)",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 18035,
        "difficulty": null
      },
      {
        "id": "1b8a707d6f193cf9c8c014e93e974b49963eb867453a167d31797e7d849b46aa",
        "domain": "numina",
        "question": "$7 \\cdot 1$ Given $n$ points on a plane, any 3 of which are the 3 vertices of a right-angled triangle, find the maximum value of $n$.\n\n untranslated text retained as requested.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 117678,
        "difficulty": null
      },
      {
        "id": "36799fe129ddb3593dd23890c9489f4d6592e4accd3b8491986851c2dee75a1a",
        "domain": "numina",
        "question": "4. First solution. Let $a_{1}, a_{2}, \\ldots, a_{15}$ be the days of the month that were sunny. Then Andrey Stepanovich will not drink a single drop from the 1st to the $a_{1}$-th day, he will drink one drop per day from the $a_{1}$-th day to the $a_{2}$-th day (including the $a_{1}$-th day but not the $a_{2}$-th day), and so on. In total, he will drink\n\n$$\n\\begin{aligned}\n& 1 \\cdot\\left(a_{2}-a_{1}\\right)+2 \\cdot\\left(a_{3}-a_{2}\\right)+\\ldots \\\\\n& \\quad \\ldots+14 \\cdot\\left(a_{15}-a_{14}\\right)+15 \\cdot\\left(30-a_{15}+1\\right)= \\\\\n& \\quad=15 \\cdot 31-\\left(a_{1}+a_{2}+\\ldots+a_{15}\\right)\n\\end{aligned}\n$$\n\nIvan Petrovich, on the other hand, will drink a number of drops equal to the sum of the numbers of all days except $a_{i}$:\n\n$$\n\\begin{aligned}\n(1+2+\\ldots+30)-\\left(a_{1}+a_{2}+\\ldots\\right. & \\left.+a_{15}\\right) \\\\\n& =15 \\cdot 31-\\left(a_{1}+a_{2}+\\ldots+a_{15}\\right)\n\\end{aligned}\n$$",
        "answer": "120",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 23294,
        "difficulty": null
      },
      {
        "id": "7449b909ce7cc4c4f3d50ad08c6f9e42807a7277172048f8b2385bb4d0341efb",
        "domain": "numina",
        "question": "C4. An integer container $(x, y, z)$ is a rectangular prism with positive integer side lengths $x, y, z$, where $x \\leq y \\leq z$. A stick has $x=y=1$; a flat has $x=1$ and $y>1$; and a box has $x>1$. There are 5 integer containers with volume 30: one stick $(1,1,30)$, three flats $(1,2,15),(1$, $3,10),(1,5,6)$ and one box $(2,3,5)$.\na. How many sticks, flats and boxes are there among the integer containers with volume 36 ?\nb. How many flats and boxes are there among the integer containers with volume 210 ?\nc. Suppose $n=p_{1}^{e_{1}} \\cdots p_{k}^{e_{k}}$ has $k$ distinct prime factors $p_{1}, p_{2}, \\ldots, p_{k}$, each with integer exponent $e_{1} \\geq 1, e_{2} \\geq 1, \\ldots, e_{k} \\geq 1$ and $k \\geq 3$. How many boxes are there among the integer containers with volume $n$ ? Express your answer in terms of $e_{1}, e_{2}, \\ldots, e_{k}$. How many boxes with volume $n=8!$ are there?",
        "answer": "280",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 110729,
        "difficulty": null
      },
      {
        "id": "3b1c46c3b4b2377e3e470c45a19c83dce8cb5f28d56e78e278d91ba1e7f87468",
        "domain": "numina",
        "question": "13. (5 points)\nThe distance from Earth to Planet Sadala is 80 light years (Note: A light year is a unit of length). Vegeta and Goku depart from Earth to Planet Sadala. Vegeta departs 1 day earlier than Goku. If Vegeta and Goku fly in a straight line, they can both fly 10 light years per day. How many days after Goku's departure will Vegeta be exactly in the middle of Goku and Planet Sadala?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101608,
        "difficulty": null
      },
      {
        "id": "6c60a14e9a370021ead674e16d7e66fea22d86cbf072b3176d2fba7901d67eef",
        "domain": "numina",
        "question": "1. Compute the sum:\n\n$$\n\\sqrt{2018-\\sqrt{2019 \\cdot 2017}}+\\sqrt{2016-\\sqrt{2017 \\cdot 2015}}+\\cdots+\\sqrt{2-\\sqrt{3 \\cdot 1}}\n$$",
        "answer": "\\frac{\\sqrt{2019}-1}{\\sqrt{2}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8530,
        "difficulty": null
      },
      {
        "id": "179ef56799651065e9ea2400adfef801e243678274a21356ee5b46b3d9344c6c",
        "domain": "numina",
        "question": "22 、A triangle with a perimeter of 36 has three sides that are all composite numbers, then the number of such triangles is. $\\qquad$",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 110071,
        "difficulty": null
      },
      {
        "id": "1970840613f6895913b3a2fd9716ad08d77e43656005f47c57542cee34a0cd4a",
        "domain": "numina",
        "question": "9.1. At the school for slackers, a competition on cheating and giving hints was organized. It is known that $75 \\%$ of the students did not show up for the competition at all, and all the rest participated in at least one of the competitions. When the results were announced, it turned out that $10 \\%$ of all those who showed up participated in both competitions, and that those who participated in the hinting competition were one and a half times more than those who participated in the cheating competition. Find the smallest possible number of students in the school for slackers.",
        "answer": "200",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9580,
        "difficulty": null
      },
      {
        "id": "e2094c063b6822e39d3c61bb1c43a7a6aa75cd5a64871bdb41dfca98228c8153",
        "domain": "numina",
        "question": "8. Find the value of $\\frac{1}{3^{2}+1}+\\frac{1}{4^{2}+2}+\\frac{1}{5^{2}+3}+\\cdots$.",
        "answer": "\\frac{13}{36}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152169,
        "difficulty": null
      },
      {
        "id": "e58fcb4220901051729511ef73e9d3c2d4dd4b1c21b630cb02c766b5c5ee852b",
        "domain": "numina",
        "question": "14. $[8]$ Danny has a set of 15 pool balls, numbered $1,2, \\ldots, 15$. In how many ways can he put the balls in 8 indistinguishable bins such that the sum of the numbers of the balls in each bin is 14,15 , or 16 ?",
        "answer": "122",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151332,
        "difficulty": null
      },
      {
        "id": "8b8ea7c78d1ac14487827e9e15bc9f0227bade997e2204d463822615b7f5b6e5",
        "domain": "numina",
        "question": "3.337. $\\sin ^{2} 70^{\\circ} \\sin ^{2} 50^{\\circ} \\sin ^{2} 10^{\\circ}=\\frac{1}{64}$.",
        "answer": "\\frac{1}{64}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 48118,
        "difficulty": null
      },
      {
        "id": "8fe92ccaf9d1c0b30e3e914a39007177868b911ed3c1ee04fc9d5ca57de6a9c3",
        "domain": "numina",
        "question": "9. In the number table below, both the upper and lower rows of numbers are arithmetic sequences. For the corresponding numbers in the upper and lower rows, the difference between the larger number and the smaller number, the smallest is $\\qquad$ _",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100745,
        "difficulty": null
      },
      {
        "id": "217d3899f28adf5290b3c5853b89837ba6ec33be8e5afa35ef416f10e47363a2",
        "domain": "numina",
        "question": "Let $P$ be a regular polygon with 2026 sides. A diagonal of $P$ is said to be good if its endpoints divide the perimeter of $P$ into two parts, each having an odd number of sides of $P$. The sides of $P$ are also considered good. Suppose that $P$ has been subdivided into triangles by 2023 diagonals that do not intersect each other inside $P$. How many, at most, such a subdivision can contain isosceles triangles with two good sides?\n\n## Solutions to the exercises\n\nTranslate the above text into English, preserving the original text's line breaks and format, and output the translation directly.",
        "answer": "1013",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 82108,
        "difficulty": null
      },
      {
        "id": "ce48bf1bcf60c39af9f02b0f0689f5df820cc684ab64666567276aedf3fe592c",
        "domain": "numina",
        "question": "9.5 In some cells of a $10 \\times 10$ table, crosses are placed such that each of them is the only one either in its row or in its column. What is the maximum number of crosses that can be in such a table? Justify your answer.",
        "answer": "18",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12936,
        "difficulty": null
      },
      {
        "id": "d869bc9921a3f4a37400e048bcad78cc196c8c9a004307c77fef9bff01cb0aa9",
        "domain": "numina",
        "question": "4. There is a four-digit integer. By adding a decimal point in front of one of its digits, and then adding this number to the four-digit integer, the sum is 2000.81. Find this four-digit number.",
        "answer": "1981",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 106525,
        "difficulty": null
      },
      {
        "id": "f1ae70532a34432e6d1b5cebded610e61cf25a6c78906744dd1f29e09fa841e6",
        "domain": "numina",
        "question": "5. (8 points) Xiao Bing participated in a strange math exam. There were a total of 100 questions, with 1 point for each correct answer, -3 points for each wrong answer, and -2 points for each unanswered question. $\\square$ It is known that Xiao Bing scored a total of 50 points. Then, the maximum number of questions Xiao Bing could have answered correctly is $\\qquad$.",
        "answer": "87",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136117,
        "difficulty": null
      },
      {
        "id": "19fcd17b8b330c4d167146e28ea1385b5967b071c71776a190fcfb5b126edb06",
        "domain": "numina",
        "question": "11. (15 points) Let the system of inequalities $\\left\\{\\begin{array}{l}x+y>0, \\\\ x-y<0\\end{array}\\right.$ represent the plane region $D$. The product of the distances from a moving point $P$ in region $D$ to the lines $x+y=0$ and $x-y=0$ is 2. The trajectory of point $P$ is denoted as curve $C$. A line $l$ passing through the point $F(2 \\sqrt{2}, 0)$ intersects curve $C$ at points $A$ and $B$. If the circle with diameter $A B$ is tangent to the $y$-axis, find the slope of line $l$.",
        "answer": "-\\sqrt{\\sqrt{2}-1}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122514,
        "difficulty": null
      },
      {
        "id": "db8a2b246aacf82b3784246992f5552abb95831d8563d0f10823d97fae78bdec",
        "domain": "numina",
        "question": "For each positive integer $n$, define the set\n$$\nP_{n}=\\left\\{n^{k} \\mid k=0,1, \\cdots\\right\\} \\text {. }\n$$\n\nFor positive integers $a$, $b$, and $c$, if there exists a positive integer $m$ such that the numbers $a-1$, $ab-12$, and $abc-2015$ (not necessarily distinct) all belong to the set $P_{m}$, then the positive integer triplet $(a, b, c)$ is called \"lucky\". Find the number of all lucky positive integer triplets.\n（Supplied by He Yijie）",
        "answer": "25",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147104,
        "difficulty": null
      },
      {
        "id": "9e6b509c2bc0d34f44ec0dabe118457eec2e8fc892c85114e732ca35d81bf16c",
        "domain": "numina",
        "question": "Marta is playing a game where she guesses a five-digit number consisting of different digits. The course of the first three rounds is as follows:\n\n| 1st round | 2 | 6 | 1 | 3 | 8 |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n| 2nd round | 4 | 1 | 9 | 6 | 2 |\n| 3rd round | 8 | 1 | 0 | 2 | 5 |\n\nThe color of the cell reveals something about the digit it contains:\n\n- a green cell means the digit is present in the guessed number and is in the correct position,\n- a yellow cell means the digit is present in the guessed number but in a different position,\n- a gray cell means the digit is not present in the guessed number.\n\nExplain whether Marta can or cannot guess the number with certainty in the next round.\n\n(J. Tkadlec)\n\nHint. Which digits can appear in the number?",
        "answer": "71284",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76888,
        "difficulty": null
      },
      {
        "id": "94500644b9079a8956d9a4f5246dd57e40487f50ef885175396486c2ef9040bd",
        "domain": "numina",
        "question": "What is the minimum length of the edge of a cube that can fit 2 unit-radius spheres, which do not have any common interior points?",
        "answer": "2+\\frac{2}{\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 165780,
        "difficulty": null
      },
      {
        "id": "7b3597e54e35f33c7439af35f40257ee741b47d28d58113cf0b56966ce5e058c",
        "domain": "numina",
        "question": "\\section*{Problem 9}\n\nEvery city in a certain state is directly connected by air with at most three other cities in the state, but one can get from any city to any other city with at most one change of plane. What is the maximum possible number of cities?",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 186982,
        "difficulty": null
      },
      {
        "id": "4ef9796a093451d364afe46cdceb847e8aefea4e999fb11e076526a26463151d",
        "domain": "numina",
        "question": "5. At each vertex of a regular $n$-gon, a number 1 or -1 is inscribed, with not all $n$ numbers being equal. The product of the numbers inscribed in any 3 consecutive vertices is -1. Determine the sum of all inscribed numbers for:\na) $n=6$;\nb) $n=2019$.\n\n## Fourth grade - B category",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 161327,
        "difficulty": null
      },
      {
        "id": "849bfab028dbd245eb6ab82513fc1cab68ff853cf06eb2584ffe3fd0feb282b8",
        "domain": "numina",
        "question": "4. Early in the morning, the pump was turned on and they started filling the pool. At 10 am, another pump was connected and by 12 pm the pool was half full. By 5 pm, the pool was full. What could be the latest time the first pump was turned on?",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 19870,
        "difficulty": null
      },
      {
        "id": "1a84632e2c853e801f6d9a590ffa54e04c962ce6e895063979df142e7707a2c8",
        "domain": "numina",
        "question": "6. In triangle $A B C$ with angle $\\angle B=120^{\\circ}$, the angle bisectors $A A_{1}, B B_{1}, C C_{1}$ are drawn. Find the degree measure of angle $C_{1} B_{1} A_{1}$.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21933,
        "difficulty": null
      },
      {
        "id": "c3230c82b3e8dd2046d7e5a8ab642eea0bfa6315f76a01f37b97b2bb47df8ac3",
        "domain": "numina",
        "question": "9-2. The school volleyball team played several matches. After they won another match, the share of victories increased by $1 / 6$. To increase the share of victories by another 1/6, the volleyball players had to win two more consecutive matches. What is the minimum number of victories the team needs to achieve to increase the share of wins by another 1/6?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16507,
        "difficulty": null
      },
      {
        "id": "3b737413ea0ec9c2437bbc26dcdf741e72ad1b7e2d52bcb51469e68eda82f670",
        "domain": "numina",
        "question": "14. (15 points) In the following addition problem, different Chinese characters can represent the same digit. How many different equations satisfy the requirement?\n\n\n\nThe translation is provided while preserving the original text's line breaks and format.",
        "answer": "100",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 108429,
        "difficulty": null
      },
      {
        "id": "96186f73fcb538c1230f9c21ed2c73ff296875c392942a3fc92d5865a8479177",
        "domain": "numina",
        "question": "Example 5 (4th American Invitational Mathematics Examination) The increasing sequence $1,3,4,9,10,12,13, \\cdots$ consists of some positive integers, which are either powers of 3 or sums of several different powers of 3. Find the 100th term of this sequence.",
        "answer": "981",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103404,
        "difficulty": null
      },
      {
        "id": "c3fdbafffd06d2374c89bd519ea35bd2d4ec2ab37914bd756de1e16edc31d357",
        "domain": "numina",
        "question": "Example 8 Find the largest positive integer $x$, such that for every positive integer $y$, $x$ divides $7^{y}+12 y-1$.\n\n---\n\nThe translation maintains the original text's format and line breaks as requested.",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 95202,
        "difficulty": null
      },
      {
        "id": "646e2af349826f20bc2790d6cb4846e6908952b7c2f4929567fd6887fb8fda23",
        "domain": "numina",
        "question": "7. [5] Find $p$ so that $\\lim _{x \\rightarrow \\infty} x^{p}(\\sqrt[3]{x+1}+\\sqrt[3]{x-1}-2 \\sqrt[3]{x})$ is some non-zero real number.",
        "answer": "\\frac{5}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 150927,
        "difficulty": null
      },
      {
        "id": "bea2f4c05189463534bcf19be9386fda6b25821672034e472f58a357ad2c5799",
        "domain": "numina",
        "question": "1. As shown in the figure, divide a line segment of length 1 into segments $x$ and $y$. Then, bend the segment of length $x$ into a semicircle $A C B$, and fold the segment of length $y$ into three sides $(B D, D E, E A)$ of a rectangle $A B D E$ to form a closed \"curved polygon\" $A C B D E A$. The maximum value of the area of this curved polygon is . $\\qquad$",
        "answer": "\\frac{1}{2(\\pi+4)}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 101508,
        "difficulty": null
      },
      {
        "id": "1c9562fe9a32d1583475139806185808b1be89227909526427b4ace8b8cac6de",
        "domain": "numina",
        "question": "Example 1. Calculate the surface area of $z=\\frac{x y}{a}$, located inside the cylinder $x^{2}+y^{2}=a^{2}, a>0$.",
        "answer": "\\frac{2\\pi}{3}^{2}(2\\sqrt{2}-1)",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31274,
        "difficulty": null
      },
      {
        "id": "f58a5916ee59c6ade1575d5aef512ae702a1ce154229b96c021868a236f9e985",
        "domain": "numina",
        "question": "13.354. When multiplying two numbers, one of which is 10 more than the other, a student made a mistake by reducing the tens digit of the product by 4. When dividing the obtained product by the smaller multiplier to check the answer, he got a quotient of 39 and a remainder of 22. Find the multipliers.",
        "answer": "3141",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49439,
        "difficulty": null
      },
      {
        "id": "9808c22d211cd458f91042fc3b98f8eb4b029453e8a8be4722935b49b3c30f32",
        "domain": "numina",
        "question": "34. Find the smallest positive integer $n$ such that $n(n+1)(n+2)$ is divisible by 247 .",
        "answer": "37",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160247,
        "difficulty": null
      },
      {
        "id": "bd52287d98524862924e7b283e1fece7b8351c081186e1d01c24cd7d4e88fccd",
        "domain": "numina",
        "question": "16. (1982 American High School Mathematics Examination) Figure 12-6 is a partial map of a city. Small rectangles represent houses and buildings, and the spaces between them represent streets. There is a student who walks from point $A$ to point $B$ every morning along the streets shown in the figure, and he can only walk east or south. At each intersection, he has a probability of $\\frac{1}{2}$ of choosing to go east or south (each choice is independent of the others). What is the probability that the student will walk through point $C$?",
        "answer": "\\frac{21}{32}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 137197,
        "difficulty": null
      },
      {
        "id": "b15c21df498723e9192623fbbd02cd03482bb85fe4599c14074fdfca291e88be",
        "domain": "numina",
        "question": "8. Given that 5 students are proficient in Chinese, Mathematics, Physics, Chemistry, and History respectively, there are 5 test papers (one each for Chinese, Mathematics, Physics, Chemistry, and History). The teacher randomly distributes one test paper to each student. The probability that at least 4 students receive a test paper that does not match their proficient subject is $\\qquad$ .",
        "answer": "\\frac{89}{120}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 110277,
        "difficulty": null
      },
      {
        "id": "a4ed4bb5dd6bc8a8a784601415d217b69f2ae9f1fcb7eab855f693799f6eb657",
        "domain": "numina",
        "question": "5. In $\\triangle A B C$, if $\\overrightarrow{A B} \\cdot \\overrightarrow{A C}=7,|\\overrightarrow{A B}-\\overrightarrow{A C}|=6$, then the maximum value of the area of $\\triangle A B C$ is",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 135102,
        "difficulty": null
      },
      {
        "id": "5136c3f332c9063e6fc50ccdf49421f6d9977331bbb182c9e1b1193d1ffbd466",
        "domain": "numina",
        "question": "2. For the quadratic trinomial $p(x)=(a+1) x^{2}-(a+1) x+2022$, it is known that $-2022 \\leq p(x) \\leq 2022$ for $x \\in[0 ; 1]$. Find the greatest possible value of $a$.",
        "answer": "16175",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8541,
        "difficulty": null
      },
      {
        "id": "e5962dec86a41af69b1b01c6001ca2f9192af844c2705a38922ef1c8b01bf76e",
        "domain": "numina",
        "question": "1. To climb from the valley to the mountain top, one must walk 4 hours on the road, and then -4 hours on the path. On the mountain top, two fire-breathing dragons live. The first dragon spews fire for 1 hour, then sleeps for 17 hours, then spews fire for 1 hour again, and so on. The second dragon spews fire for 1 hour, then sleeps for 9 hours, then spews fire for 1 hour again, and so on. It is dangerous to walk on both the road and the path during the first dragon's fire-spewing, while it is dangerous to walk only on the path during the second dragon's fire-spewing. Both dragons start spewing fire simultaneously at midnight. Is it possible to safely climb from the valley to the mountain top and return back? (8 points)",
        "answer": "38",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 7499,
        "difficulty": null
      },
      {
        "id": "c1240299ec5b783e7d6a5f60390a40fd88ef218ffb45c6fc5869d75061d54191",
        "domain": "numina",
        "question": "3-2. Two identical polygons were cut out of cardboard, aligned, and pierced with a pin at some point. When one of the polygons is rotated around this \"axis\" by $25^{\\circ} 30^{\\prime}$, it aligns again with the second polygon. What is the smallest possible number of sides of such polygons?",
        "answer": "240",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 26234,
        "difficulty": null
      },
      {
        "id": "50fb86557bff328cb501a87bc39636780500b51c385c9a58af22e774440b4ce3",
        "domain": "numina",
        "question": "4. If $a+b+c=1$, and $a, b, c \\in \\mathbf{R}^{+}$, then the minimum value of $\\sqrt{a^{2}+b^{2}}+\\sqrt{b^{2}+c^{2}}+\\sqrt{c^{2}+a^{2}}$ is $\\qquad$ .",
        "answer": "\\sqrt{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 93658,
        "difficulty": null
      },
      {
        "id": "9f2054ebd50c8b23f90313f3e2c6a3564110cf6c57fdbdb55a8b7efba648f6bc",
        "domain": "numina",
        "question": "99. Person A departs from point A to point B, while at the same time, person B departs from point B to point A. After 12 minutes, the two have passed their meeting point and are 100 meters apart. It is known that person A takes 20 minutes to complete the journey, and person B walks at a speed of 65 meters per minute. Therefore, the distance between points A and B is ______ meters. $\\qquad$",
        "answer": "1700",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128441,
        "difficulty": null
      },
      {
        "id": "be19f46c6df3bc97870a6ca9186fe9f5efb8c4c126ca97885584f3b95c313cf4",
        "domain": "numina",
        "question": "8. A certain intelligence station has four different passwords $A, B, C, D$, and uses one of them each week. Each week, one of the three passwords not used in the previous week is chosen with equal probability. If the first week uses password $A$, what is the probability that the seventh week also uses password $A$? (Express your answer as a simplified fraction)\n\nFill in the blank: $\\qquad$ .",
        "answer": "\\frac{61}{243}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 139729,
        "difficulty": null
      },
      {
        "id": "647198b757f4317216e9b2941c519269db1aa66ef581b748f351900ac1555c21",
        "domain": "numina",
        "question": "16. Fill in each square in $\\square \\square+\\square \\square=\\square \\square \\square$ with a digit from $0, 1, 2, \\cdots \\cdots, 9$ (digits in the squares can be the same, and the leading digit of any number cannot be 0), so that the equation holds. \n\nThere are",
        "answer": "4860",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 124167,
        "difficulty": null
      },
      {
        "id": "e9146285978a9024aceb1a9ec8e3e03f380dab02320f5b05e7680a444f9559bd",
        "domain": "numina",
        "question": "51. In a box, there are 250 light bulbs, of which 100 are 100W, 50 are 60W, 50 are 25W, and 50 are 15W. Calculate the probability that the power of any randomly selected light bulb will not exceed 60W.",
        "answer": "\\frac{3}{5}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32271,
        "difficulty": null
      },
      {
        "id": "db0f5a2f051357192c1ccacd31e5bdcfdcd878cd8823a8b6243892297b538a79",
        "domain": "numina",
        "question": "5. Let $C D$ be a chord of a circle $\\Gamma_{1}$ and $A B$ a diameter of $\\Gamma_{1}$ perpendicular to $C D$ at $N$ with $A N>N B$. A circle $\\Gamma_{2}$ centred at $C$ with radius $C N$ intersects $\\Gamma_{1}$ at points $P$ and $Q$, and the segments $P Q$ and $C D$ intersect at $M$. Given that the radii of $\\Gamma_{1}$ and $\\Gamma_{2}$ are 61 and 60 respectively, find the length of $A M$.",
        "answer": "78",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160630,
        "difficulty": null
      },
      {
        "id": "5368904d597872b1456b10c6d5b80489c121dd1fb02751ac06a3995ea34c4757",
        "domain": "numina",
        "question": "7. (10 points) The calculation result of the expression $1007 \\times \\frac{1 \\frac{3}{4} \\div \\frac{3}{4}+3 \\div 2 \\frac{1}{4}+\\frac{1}{3}}{(1+2+3+4+5) \\times 5-22} \\div 19$ is",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138794,
        "difficulty": null
      },
      {
        "id": "8b7e980efbca67f4dfcbaf940f3396fda25a905ef2b6408fc9bc490b31dac60f",
        "domain": "numina",
        "question": "1. (7 points) 1. Calculate: $\\left(7 \\frac{4}{25}+8.6\\right) \\div\\left[\\left(4 \\frac{5}{7}-0.005 \\times 900\\right) \\div \\frac{6}{7}\\right]=$",
        "answer": "63.04",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92311,
        "difficulty": null
      },
      {
        "id": "1b934cdac6c324ccb2c911fe01ffe735bf68e2fa123578b73f4f6e8b8a7204a5",
        "domain": "numina",
        "question": "19. Given an arbitrary triangle. On each side of the triangle, 10 points are marked. Each vertex of the triangle is connected by segments to all the marked points on the opposite side. Into what maximum number of parts could the segments divide the triangle?",
        "answer": "331",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 162147,
        "difficulty": null
      },
      {
        "id": "47069b4da61ac2a67d031c76f4e5b5ce0991e1432362818ddea57b7c26834632",
        "domain": "numina",
        "question": "## Task A-2.5.\n\nIn a coordinate system, all integer points $(x, y)$ are marked where $1 \\leqslant x \\leqslant$ 200 and $1 \\leqslant y \\leqslant 100$, a total of 20000 points. How many line segments of length $\\sqrt{5}$ have their endpoints at the marked points?",
        "answer": "78208",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89573,
        "difficulty": null
      },
      {
        "id": "1e02b45d55799b590ea52c362e01ef3c820d0bbeea3d8fc3f5c69c938e5e5631",
        "domain": "numina",
        "question": "10. The graph on the right shows the fuel consumption per 100 kilometers for four different types of cars: $U, V, W, X$. If each car has 50 liters of fuel, what is the maximum total distance that these four cars can travel in kilometers?",
        "answer": "1950",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 108211,
        "difficulty": null
      },
      {
        "id": "5aaa561ec3d124f5fade2405410e3e0ddd16ef498d7431ca6623513d47922acb",
        "domain": "numina",
        "question": "A10. If you open a book and add the numbers that mark the left and right page, you get 21. What number do you get if you multiply the same two numbers?\n(A) 100\n(B) 121\n(C) 420\n(D) 110\n(E) 200\n\n# 4th Regional Mathematics Knowledge Competition for Vocational School Students\n\nMarch 31, 2004\n\n## II. Part: LONGER PROBLEMS\n\nInstruction: Solve problems B1 to B4 of the second part on the provided paper, where you write down the complete calculations. Read each problem carefully and answer the questions asked. The solution to each of these problems will be rated from 0 to 5 points.\n\nNote that you have 90 minutes to solve the problems of the first and second parts.",
        "answer": "110",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 163349,
        "difficulty": null
      },
      {
        "id": "077eef4c44465754c844c7247cc41a391a17190fb895546f417e49add91ee9ab",
        "domain": "numina",
        "question": "1. In the wallet, there are six different coins of cents and euros (5 cents, 10 cents, 20 cents, 50 cents, $1 €$ and $2 €$). Three coins are drawn simultaneously and without looking. In how many different ways can the coins be drawn so that their total value is greater than 85 cents?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86589,
        "difficulty": null
      },
      {
        "id": "2d5d51bc063111290f64327847d42ea238b9f9844fdeed10c2b242099e68ee3c",
        "domain": "numina",
        "question": "3. Aroon's PIN has four digits. When the first digit (reading from the left) is moved to the end of the PIN, the resulting integer is 6 less than 3 times Aroon's PIN. What could Aroon's PIN be?",
        "answer": "2856",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154949,
        "difficulty": null
      },
      {
        "id": "ae95c347a5eeec235ae3d0294b5b6ca03c4053513537df2328fccdaf48928307",
        "domain": "numina",
        "question": "11. (12 points) Fill in the blanks with numbers $1-6$, so that the numbers in each row and column do not repeat, the numbers on both sides of the black dot are in a two-fold relationship, and the numbers on both sides of the white dot have a difference of 1. Then the five-digit number formed by the first 5 numbers from left to right in the fourth row is $\\qquad$\n\nFill in the blanks with numbers $1-6$, so that the numbers in each row and column do not repeat, the numbers on both sides of the black dot are in a two-fold relationship, and the numbers on both sides of the white dot have a difference of 1. Then the five-digit number formed by the first 5 numbers from left to right in the fourth row is $\\qquad$",
        "answer": "21436",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 94545,
        "difficulty": null
      },
      {
        "id": "7df6cd1a954c65b9bd81c9d213d3535118291f04eb554846ffed64477e36cd40",
        "domain": "numina",
        "question": "10. Let the integer $n \\geqslant 4, A=\\left\\{a_{1}, a_{2}, \\cdots, a_{n}\\right\\}$ be a set of integers, satisfying $2023 \\in A$, and the absolute difference between any two elements of $A$ is a prime number. Then the number of such sets $A$ is $\\qquad$.",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136807,
        "difficulty": null
      },
      {
        "id": "366cf4953115941b92cfad9d715f9a41ac27e895e99c02e4e264dafcc42f4dbc",
        "domain": "numina",
        "question": "49. A ship sails downstream from Port A to Port B, and upon arrival, immediately returns upstream to Port A, taking a total of 12 hours. It is known that the ship travels 16 kilometers more per hour downstream than upstream, and it also travels 80 kilometers more in the first 6 hours than in the last 6 hours. Therefore, the distance between Port A and Port B is $\\qquad$ kilometers.",
        "answer": "280",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 137345,
        "difficulty": null
      },
      {
        "id": "984c5dc798afc79aa92f53f4b1f380be690dde7e85def20e7eab83e603a1b5d2",
        "domain": "numina",
        "question": "9. Two people, A and B, agree to meet at a certain place within 10 days. They agree that the one who arrives first will wait for the other, but can leave after waiting for 3 days. If they are equally likely to arrive at the destination within the limit, then the probability that these two people will meet is $\\qquad$ .",
        "answer": "\\frac{51}{100}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 142938,
        "difficulty": null
      },
      {
        "id": "37e78f860c13e66305460e34e5d31d19b6bad8c7098b72100dce9834d258b34b",
        "domain": "numina",
        "question": "37. Find the remainder when $1^{2017}+2^{2017}+3^{2017}+4^{2017}+5^{2017}$ is divided by 5. (where $a^{2017}$ denotes 2017 instances of $a$ multiplied together)",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 132451,
        "difficulty": null
      },
      {
        "id": "e7194566986e33aa93344af35ea637aee6bb9f03b4f3aa6abb70a1d77ab80f0e",
        "domain": "numina",
        "question": "7. Let $A B C$ be an obtuse triangle with circumcenter $O$ such that $\\angle A B C=15^{\\circ}$ and $\\angle B A C>90^{\\circ}$. Suppose that $A O$ meets $B C$ at $D$, and that $O D^{2}+O C \\cdot D C=O C^{2}$. Find $\\angle C$.",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151462,
        "difficulty": null
      },
      {
        "id": "7064167d93942316e500bbfdf8737526119aa8f76966220a4dc6d83f1c6d8ce4",
        "domain": "numina",
        "question": "1. Calculate $2015-5 \\cdot(25 \\cdot 13+13+3 \\cdot 25-10)$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86667,
        "difficulty": null
      },
      {
        "id": "219b6e5eb41fe48b677fb35c6e73dadbe870e6d3f9eeb102d79fe053e46d6b7e",
        "domain": "numina",
        "question": "6.31 Team A and Team B each send out 7 players to participate in a Go relay match according to a pre-arranged order. Both sides start with the No. 1 player competing, the loser is eliminated, and the winner then competes with the No. 2 player from the losing side, $\\cdots$, until all players from one side are eliminated, and the other side wins, forming a match process. How many different possible match processes are there?",
        "answer": "3432",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 128686,
        "difficulty": null
      },
      {
        "id": "9c6b633a32b9dd77d335b6ad327df7da273f037f99b8254cd0ac51ae91d05799",
        "domain": "numina",
        "question": "Task 2. Lёnya has cards with digits from 1 to 7. How many ways are there to glue them into two three-digit numbers (one card will not be used) so that their product is divisible by 81, and their sum is divisible by 9?",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4560,
        "difficulty": null
      },
      {
        "id": "d1796ff623ada762883f2d76e148300d20943c2b4a5e1591e5093e44cd371a34",
        "domain": "numina",
        "question": "Example 2 Given that $x, y$ are both in the interval $(-2,2)$, and $x y=-1$, then what is the minimum value of the function $u=\\frac{4}{4-x^{2}}+\\frac{9}{9-y^{2}}$?",
        "answer": "\\frac{12}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102201,
        "difficulty": null
      },
      {
        "id": "f6d874205f7088b4e36a5556c58a7d6819ded2393197a7fb6413918ba3ffccee",
        "domain": "numina",
        "question": "5. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=a>2, a_{2017}=$ 2017, and for any positive integer $n, a_{n+1}=a_{n}^{2}-2$. Then $\\left[\\frac{\\sqrt{a-2}}{10^{6}} a_{1} a_{2} \\cdots a_{2017}\\right]=$ $\\qquad$, where $[x]$ denotes the greatest integer not exceeding the real number $x$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93178,
        "difficulty": null
      },
      {
        "id": "1a1dcaf5fc10920ff004199da3dbe4529381bd1d6699d9540ac94535eff486db",
        "domain": "numina",
        "question": "\\section*{Problem 1 - V01001}\n\nThe digit sum of a two-digit number is 9. If you multiply the number by 5 and subtract 9 from the product, you get a two-digit number with the same digits in reverse order. What is the two-digit number?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184819,
        "difficulty": null
      },
      {
        "id": "a4e7111bcd3ac5fcbe24ee73759b18f90f55445c4478628457f3df934ca56944",
        "domain": "numina",
        "question": "Problem 2. Anya, Borya, Vika, and Gena are on duty at school for 20 days. It is known that exactly three of them are on duty each day. Anya was on duty 15 times, Borya - 14 times, Vika - 18 times. How many times was Gena on duty",
        "answer": "13",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18324,
        "difficulty": null
      },
      {
        "id": "42de31c3714f88a13ba4d5043160c51ac3fe4a4c6e3f517350a58848a456bd0f",
        "domain": "numina",
        "question": "8. Let $x_{i} \\in\\{1,2, \\cdots, n\\}, i=1,2, \\cdots, n$ satisfy $\\sum_{i=1}^{n} x_{i}=\\frac{n(n+1)}{2}, x_{1} x_{2} \\cdots x_{n}=n$!. Then, the largest positive integer $n$ for which $x_{1}, x_{2}, \\cdots, x_{n}$ must be a permutation of $1,2, \\cdots, n$ is $\\qquad$.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 114045,
        "difficulty": null
      },
      {
        "id": "8738e37ec0716140102a26642ad20f3e8f871109a779d1606d24f2f193d44a6b",
        "domain": "numina",
        "question": "G3.1 Simplify $\\sqrt{3+\\sqrt{5}}+\\sqrt{3-\\sqrt{5}}$.",
        "answer": "\\sqrt{10}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153388,
        "difficulty": null
      },
      {
        "id": "81b25b6973758b32626c4617b38a76d55a6c94e5eed8b66c560e1473196945ee",
        "domain": "numina",
        "question": "## Task B-1.6.\n\nA student has passed 31 exams over the course of a five-year study. Each year, he took more exams than the previous year, and in the fifth year, he took three times as many exams as in the first year. How many exams did the student take in the fourth year?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87796,
        "difficulty": null
      },
      {
        "id": "399552b4bc7308aac96662cac6877974eee51ced789b8c91bc53889d3f4de20d",
        "domain": "numina",
        "question": "Task 1. In the first week, a squirrel collected 84 nuts, in the second week 96 nuts, and in the third week 65 nuts. Another squirrel in the fourth week collected three times more nuts than the first squirrel collected in the second and third weeks combined. How many nuts did the two squirrels collect in total over the four weeks?",
        "answer": "728",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189014,
        "difficulty": null
      },
      {
        "id": "df4118c38cc9fb54231125834358387a5b58bc2130c7a14715a0cf2a1993111b",
        "domain": "numina",
        "question": "Let's determine the largest of the $A$ numbers for which the following statement holds. No matter how we write down the first 100 positive integers, there will always be ten consecutive numbers in the sequence whose sum is at least $A$.",
        "answer": "505",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 168352,
        "difficulty": null
      },
      {
        "id": "a3b5d47b51420f870d40e782fe6ee875d31cdb55040c9a1bbdb0914cf5b4a38f",
        "domain": "numina",
        "question": "# Problem 3.\n\nB-1\n\nAt the time when a lion cub, located 6 minutes away, set off for a drink, the second, having already quenched its thirst, headed back along the same path at 1.5 times the speed of the first. At the same time, a tortoise, located 32 minutes away, set off along the same path for a drink. After some time, the first lion cub stepped on the tortoise, and some time later, the second lion cub did the same. 28 minutes and 48 seconds after the second incident, the tortoise reached the water hole. How many minutes passed between the two incidents, given that all three moved at constant speeds?",
        "answer": "2.4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 5901,
        "difficulty": null
      },
      {
        "id": "c775996479dd6451cdc57f2a1e359f6fe6c642548ea2e5ed61258b827c5d0cad",
        "domain": "numina",
        "question": "8. A mob of 2015 kangaroos contains only red and grey kangaroos. One grey kangaroo is taller than exactly one red kangaroo, one grey kangaroo is taller than exactly three red kangaroos, one grey kangaroo is taller than exactly five red kangaroos and so on with each successive grey kangaroo being taller than exactly two more red kangaroos than the previous grey kangaroo. The final grey kangaroo is taller than all the red kangaroos. How many grey kangaroos are in the mob?",
        "answer": "672",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156416,
        "difficulty": null
      },
      {
        "id": "ee1c2359fa338a72c0288777ba4f5cccca9c9e8298bf439edc34f9a436587db3",
        "domain": "numina",
        "question": "2. (8 points) There is a special calculator. When a number is input, the calculator first multiplies this number by 3, then reverses the digits of the result, and finally adds 2 to display the final result. Xiao Ming input a four-digit number, and the displayed result was 2015. What is the four-digit number that Xiao Ming input? $\\qquad$ .",
        "answer": "1034",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 128616,
        "difficulty": null
      },
      {
        "id": "041d3c4633236ea0c20c347261c56844cb70d9ceccad71d292f6b7f70d8a40d2",
        "domain": "numina",
        "question": "Twelve points are marked on a sheet of graph paper, as shown in the figure. What is the maximum number of squares that can be formed by connecting four of these points?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_8935ac0233b397f8e8f1g-2.jpg?height=320&width=326&top_left_y=768&top_left_x=1393)",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74025,
        "difficulty": null
      },
      {
        "id": "3ba568c2a5fa0805754caadafbc43a1d6679791fbc2c1de89a8078357972b04e",
        "domain": "numina",
        "question": "Subject 2. Solve in integers the equation: $2^{x+1}=x^{2}+x+1$.\n\nMathematical Gazette 1987",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 179066,
        "difficulty": null
      },
      {
        "id": "7faf48bddde170b0f05244b0fbddda663b513196e75fa744b5cce42ee52d60ad",
        "domain": "numina",
        "question": "I1.2 If $a$ copies of a right-angled isosceles triangle with hypotenuse $\\sqrt{2} \\mathrm{~cm}$ can be assembled to form a trapezium with perimeter equal to $b \\mathrm{~cm}$, find the least possible value of $b$. (give the answer in surd form).",
        "answer": "2+4\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152994,
        "difficulty": null
      },
      {
        "id": "256cf8fb5e48b91f2735eab7a2f715e3bd095b1c806d98aed89855f79c892656",
        "domain": "numina",
        "question": "G5.4 If $\\cos ^{6} \\theta+\\sin ^{6} \\theta=0.4$ and $d=2+5 \\cos ^{2} \\theta \\sin ^{2} \\theta$, find the value of $d$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152928,
        "difficulty": null
      },
      {
        "id": "7d9d5ea3f99600ea717cf9e5b8d197fb107dfa3653f88f71a1d27450e18af4f6",
        "domain": "numina",
        "question": "The bases of the trapezoid are 8 and 2. The angles adjacent to the larger base are each $45^{\\circ}$. Find the volume of the solid formed by rotating the trapezoid around the larger base.",
        "answer": "36\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59271,
        "difficulty": null
      },
      {
        "id": "178d4cfa52021505e82687d5d5ca6c420776b2e9233d9cc561298ab4f5258a29",
        "domain": "numina",
        "question": "7.2. A bus with programmers left Novosibirsk for Pavlodar. When it had traveled 70 km, Pavel Viktorovich set off from Novosibirsk in a car along the same route, and caught up with the programmers in Karasuk. After that, Pavel drove another 40 km, while the bus traveled only 20 km in the same time. Find the distance from Novosibirsk to Karasuk, if both the car and the bus traveled at constant speeds. (Provide a complete solution, not just the answer.)",
        "answer": "140",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9618,
        "difficulty": null
      },
      {
        "id": "183cdf7fd1ee9eb2a63eba084241a03b488866204343a0cecfdefcd427224fc0",
        "domain": "numina",
        "question": "G4.2 If $x=\\frac{\\sqrt{5}+1}{2}$ and $y=\\frac{\\sqrt{5}-1}{2}$, determine the value of $x^{3} y+2 x^{2} y^{2}+x y^{3}$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153374,
        "difficulty": null
      },
      {
        "id": "4d7f559280bf75d53f197e17ea61b1833bfb71c6b5b00688a1f96b49b2f7796e",
        "domain": "numina",
        "question": "25. The area of a square is 40 square centimeters, then the area of its circumscribed circle is $\\qquad$ square centimeters. ( $\\pi$ is taken as 3.14)",
        "answer": "62.8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 130147,
        "difficulty": null
      },
      {
        "id": "9a30fb7568b151edcbc64733e4c91b8b83ce126f060c01f0bdf5189ba31bcc36",
        "domain": "numina",
        "question": "133. Digits and Prime Numbers. Using each of the nine digits once and only once, form prime numbers (numbers that do not divide evenly by any integer except 1 and themselves) whose sum is the smallest.\n\nFor example, four prime numbers\n\n61\n\n+283\n\n47\n\n59\n\n450\n\ncontain all nine digits once and only once, and their sum is 450, but it can be significantly reduced. This is a very simple puzzle.",
        "answer": "207",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41049,
        "difficulty": null
      },
      {
        "id": "9339b14f300f559cc690a30cf49475e06c19ddf085d950e3069d0ab04e329489",
        "domain": "numina",
        "question": "II. (40 points) Let $p$ be a prime number, and the sequence $\\left\\{a_{n}\\right\\}(n \\geqslant 0)$ satisfies $a_{0}=0, a_{1}=1$, and for any non-negative integer $n$, $a_{n+2}=2 a_{n+1}-p a_{n}$. If -1 is a term in the sequence $\\left\\{a_{n}\\right\\}$, find all possible values of $p$.\n\n保留了原文的换行和格式。",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140985,
        "difficulty": null
      },
      {
        "id": "71f1b716480e648b9cf98abd17e2fdda14f0d5be36ba29223c7af232fb05ce77",
        "domain": "numina",
        "question": "Four consecutive integers have a product that is a perfect square. What are these numbers?",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 168281,
        "difficulty": null
      },
      {
        "id": "7450ab3c81b2a9198b2f5d17af076a5e05e1f3a5b147c6c6bebf39578052a035",
        "domain": "numina",
        "question": "We drew a 13 cm and a $35 \\mathrm{~cm}$ side square on a grid paper, with each side lying on a grid line of the paper, and the distance between the lines is $1 \\mathrm{~cm}$ in both directions. We want to cut these two squares into pieces so that we can form three new squares with side lengths of 17, 23, and $24 \\mathrm{~cm}$. The cutting must be done along the grid lines. Provide a dissection that results in as few pieces as possible. (For example, it is sufficient to submit a $1: 4$ scale drawing of the dissection and the reassembly, indicating the number of pieces and briefly explaining why the given solution is better than other attempts.)\n\n\n\n\n\nTranslating the text as requested, while preserving the original formatting and line breaks.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 164360,
        "difficulty": null
      },
      {
        "id": "5f4fbc51812f65de667cbe7a4e16262c409b777376a689968e38636657041c55",
        "domain": "numina",
        "question": "4. Let $A=\\{(\\alpha, \\beta, \\gamma) \\mid \\alpha+\\beta \\sin x+\\gamma \\cos x=0, x \\in(-\\infty,+\\infty)\\}$, then the number of elements in the set $A$ is $m=$\n$\\qquad$ .",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105108,
        "difficulty": null
      },
      {
        "id": "4f99b4f5c6e3b94f5d3c3350eac13fc226eb0744b6a71806c2f4529b3e33f1e3",
        "domain": "numina",
        "question": "329. Find the value of the arbitrary function $f(x)=\\sin ^{4} x-$ $-\\cos ^{4} x$ at $x=\\pi / 12$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32008,
        "difficulty": null
      },
      {
        "id": "113de6bd4c8d49838b886e417d315d482b8839bf2b6b3c28b72278b5b62973ef",
        "domain": "numina",
        "question": "## Problem 2\n\nFind all real roots of the quartic $\\mathrm{x}^{4}-(2 \\mathrm{~N}+1) \\mathrm{x}^{2}-x+\\mathrm{N}^{2}+N-1=0$ correct to 4 decimal places, where $\\mathrm{N}=$ $10^{10}$.",
        "answer": "99999.9984100000.0016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 187152,
        "difficulty": null
      },
      {
        "id": "b5e02fa22c21d7de8b2a63d1da3b7eb4ec0363aed9b13a78b7882af9dd246cc3",
        "domain": "numina",
        "question": "10,11\n\nFind the length of the shortest path on the surface of a unit regular tetrahedron between the midpoints of its opposite edges.\n\n#",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66646,
        "difficulty": null
      },
      {
        "id": "049daf8c53d506ad0537808b84864134a5728847416e7dd40e1fd7a6be357c58",
        "domain": "numina",
        "question": "7.1. Given two different geometric progressions, the first terms of which are equal to 1, and the sum of the denominators is 3. Find the sum of the fifth terms of these progressions, if the sum of the sixth terms is 573. If the answer to the question is not unique, specify the sum of all possible values of the desired quantity.",
        "answer": "161",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6365,
        "difficulty": null
      },
      {
        "id": "932469bdeb68c305d46096216af8cfbab45dbbae42c4f0fa2a97d359d2c8d5e9",
        "domain": "numina",
        "question": "5. If $a, b$ are nonzero real numbers such that $a^{2}+b^{2}=8 a b$, find the value of $\\left|\\frac{a+b}{a-b}\\right|$.",
        "answer": "\\frac{\\sqrt{15}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152400,
        "difficulty": null
      },
      {
        "id": "32474b0f572c164dd3cc239d54f1dd083a0f9c4cf060656caf6f26b2ef819459",
        "domain": "numina",
        "question": "All the points on the line with equation $y=3 x+6$ are translated up 3 units, then translated left 4 units, and then reflected in the line with equation $y=x$. Determine the $y$-intercept of the resulting line.",
        "answer": "-7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194373,
        "difficulty": null
      },
      {
        "id": "118439ce4473e6e9366dcc63191868051261674cf58d88e10e8d5da98cfed997",
        "domain": "numina",
        "question": "10.3. In the distant times of stagnation in the Soviet Union, 15 and 20 kopeck coins were in circulation. Schoolboy Valera had a certain amount of money only in such coins. Moreover, the number of 20 kopeck coins was greater than the number of 15 kopeck coins. Valera spent one-fifth of all his money, paying two coins for a movie ticket. He then spent half of the remaining money on lunch, paying with three coins. How many coins of each denomination did Valera have initially? Justify your answer.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 18382,
        "difficulty": null
      },
      {
        "id": "f3c9c30644f8280b6cca58bcbbb4b078f804dcbe7304688da0f69719f93cf14f",
        "domain": "numina",
        "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$M_{1}(7 ; 2 ; 4)$\n\n$M_{2}(7 ;-1 ;-2)$\n\n$M_{3}(-5 ;-2 ;-1)$\n\n$M_{0}(10 ; 1 ; 8)$",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 47200,
        "difficulty": null
      },
      {
        "id": "f496eb3242f60c7f07ac297161aedceae051b0c2edae8f84d9b90121dfde2012",
        "domain": "numina",
        "question": "1.2. Find the greatest negative root of the equation $\\sin \\pi x = -\\sqrt{3} \\cos \\frac{\\pi x}{2}$.",
        "answer": "-\\frac{2}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4937,
        "difficulty": null
      },
      {
        "id": "af8af11ca3f5f54895e5730e6701c872116cfc7136df652a359666cf381b73a4",
        "domain": "numina",
        "question": "The function $f(x)$ satisfies for any positive numbers $x, y$: $f(x y)=f(x)+f(y)$, and $f(2)=1$, then the value of $f\\left(\\frac{1}{64}\\right)$ is $\\qquad$ .",
        "answer": "-6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120254,
        "difficulty": null
      },
      {
        "id": "628c13a15dc7beb1d288ac78d766ed1b67f7339e944292f373e8ce3df55f8bec",
        "domain": "numina",
        "question": "## Task B-4.4.\n\nGrandpa Ante, looking for a way to entertain his grandchildren Iva and Mato, found three cards. On the first card, one side has the number 1, and the other side has the number 4. On the second card, one side has the number 2, and the other side has the number 4. On the third card, one side has the number 3, and the other side also has the number 4. Grandpa Ante immediately knew how to keep the children occupied for a long time. He told the younger one, Mato, to arrange the three cards in a row, and the schoolboy Iva had to write down the three-digit numbers that Mato forms by arranging the cards and then calculate the sum of all the recorded numbers. Mato randomly picks a card and its side. How many different three-digit numbers can Ivo write down at most, and what is the sum of these numbers?",
        "answer": "10434",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88774,
        "difficulty": null
      },
      {
        "id": "d4d5872409a64dc896c13d551207467a1b29c694d759194c96f798ae868aa04c",
        "domain": "numina",
        "question": "2. Find the number of positive integer solutions $(x, y, z)$ that satisfy the following system of equations:\n\\[\n\\left\\{\\begin{array}{l}\nx^{2}+y^{2}=2 z^{2}, \\\\\nz<y \\leqslant z+50\n\\end{array}\\right.\n\\]",
        "answer": "131",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114574,
        "difficulty": null
      },
      {
        "id": "f6d477815f39323f399de22e63c9a02d706b55c20467b0d7a21c271b819758b9",
        "domain": "numina",
        "question": "Example 2. Into how many parts should the interval of integration be divided to calculate $\\int_{2}^{7} \\frac{d x}{\\sqrt{x+2}}$ with an accuracy of 0.1?",
        "answer": "8",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32871,
        "difficulty": null
      },
      {
        "id": "b09762f2482fd778bbb9987785133fada8e57f60fe829725357a8ea87031f3b6",
        "domain": "numina",
        "question": "8. The number of different shapes of ellipses with eccentricity $e=\\log _{p} q$ (where $p, q$ are positive integers not exceeding 9) is\n\n保留了源文本的换行和格式。",
        "answer": "26",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113112,
        "difficulty": null
      },
      {
        "id": "749c26cd4f273f92dcda2c0671cdb9880b30a1b30db3a915370dc83327e28de9",
        "domain": "numina",
        "question": "3. Let $A, B, C$ be three distinct points on a circle, $P$ be the area of triangle $ABC$, and $P_{1}$ be the area of the triangle determined by the tangents to the circle at points $A, B, C$. Determine the limit of the ratio $P_{1}: P$ as point $A$ is fixed and points $B$ and $C$ approach $A$ along the circle such that $B \\neq C$.",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189579,
        "difficulty": null
      },
      {
        "id": "391c56af0491b3a1b8b08f5acc2bf216a64fd063fa8da581582d5c54513152a8",
        "domain": "numina",
        "question": "2.145. $\\frac{1-a x}{1+a x} \\cdot \\sqrt{\\frac{1+b x}{1-b x}} ; \\quad x=\\frac{1}{a} \\cdot \\sqrt{\\frac{2 a-b}{b}} ; \\quad 0<\\frac{b}{2}<a<b$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50492,
        "difficulty": null
      },
      {
        "id": "b9ceac0b6182a961ae4d2dd82ac219e7010d9add5e354596bd24dbf9375b475d",
        "domain": "numina",
        "question": "10.16 There is a five-digit positive odd number $x$. In $x$, all 2s are replaced with 5s, and all 5s are replaced with 2s, while other digits remain unchanged, resulting in a new five-digit number, denoted as $y$. If $x$ and $y$ satisfy the equation $y=2(x+1)$, what is the value of $x$?\n(China Junior High School Mathematics League, 1987)",
        "answer": "29995",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112478,
        "difficulty": null
      },
      {
        "id": "a0196b4f420ae66561851b29801673bf3b202f7ae7904d87bb8e02dcf8fa6603",
        "domain": "numina",
        "question": "3. In tetrahedron $ABCD$, $\\triangle ADB$ is an isosceles right triangle, $AD=1, \\angle ADB=90^{\\circ}, \\angle BDC=$ $\\angle ADC=60^{\\circ}$. Then the distance between the skew lines $AB$ and $CD$ is $\\qquad$",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 135543,
        "difficulty": null
      },
      {
        "id": "f56b8ecac4f96eb916fe19f4284f0e21896e0bb78ef904fe199df4d4e4bce978",
        "domain": "numina",
        "question": "1. Find the maximum value of the function $f(x)=(\\sin x)^{\\cos x}$ on the interval $\\left[0, \\frac{\\pi}{2}\\right]$.",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 161429,
        "difficulty": null
      },
      {
        "id": "3585134602e75b36ffc00a8c386c8f472debe73b29c1d61cb32be5a88d8fa544",
        "domain": "numina",
        "question": "6. 10 students (one is the team leader, 9 are team members) formed a team to participate in a math competition and won the first place. The organizing committee decided to award each team member 200 yuan, and the team leader received 90 yuan more than the average prize of the entire team of 10 members. Therefore, the team leader's prize is $\\qquad$ yuan.",
        "answer": "300",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127840,
        "difficulty": null
      },
      {
        "id": "a60c1c3fd91bdaaed3fe421d3a908c0868a24960d3a8f588533b36e85f3a1e8c",
        "domain": "numina",
        "question": "1. From a square steel sheet with a side of 1 meter, identical rectangular isosceles triangles are cut from each of the four corners, leaving a smaller square. Determine its mass if the thickness of the sheet is 3 mm and the density of steel is 7.8 $g / \\mathrm{cm}^{3}$.",
        "answer": "11.7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11247,
        "difficulty": null
      },
      {
        "id": "0d5e47745708ecf03599cbd95cbccf726f50949a83521b2e892990edcac4264f",
        "domain": "numina",
        "question": "In a right trapezoid $A B C D(A B \\| C D)$ the angle at vertex $B$ measures $75^{\\circ}$. Point $H$ is the foot of the perpendicular from point $A$ to the line $B C$. If $B H=D C$ and $A D+A H=8$, find the area of $A B C D$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 197642,
        "difficulty": null
      },
      {
        "id": "092fa30da3ff9c4a9e951ac78b0940689757e35ae2e7c48507bf11df78415e31",
        "domain": "numina",
        "question": "81. Determine the sum of the coefficients of the polynomial that results from expanding and combining like terms in the expression $\\left(1+x-3 x^{2}\\right)^{1965}$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 28473,
        "difficulty": null
      },
      {
        "id": "9f431069031a9114023ad6cf790e5a1037bd604271fd36a59917a8608c06ac4e",
        "domain": "numina",
        "question": "14. (15 points) On the blackboard, there are 5 natural numbers: 1, 3, 5, 7, 9. One operation involves arbitrarily selecting two numbers from the blackboard, erasing them, and writing their sum on the blackboard, while writing their product on a piece of paper. After 4 such operations, there is only 1 number left on the blackboard, and 4 numbers are written on the paper. Find the sum of these 4 numbers.",
        "answer": "230",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 139558,
        "difficulty": null
      },
      {
        "id": "3e2fc7612c40ad4fe95058eb0e27507c38a7957287997ad14e2f4b68fddceef3",
        "domain": "numina",
        "question": "13.031. A group of students went on a hike through the Moscow region during their vacation. The first 30 km they walked, $20 \\%$ of the remaining part of the route they traveled by raft along the river, and then they walked again, covering a distance 1.5 times greater than the distance they traveled by raft. The remaining part of the journey was covered in 1 hour 30 minutes by hitching a ride on a passing truck, which was traveling at a speed of 40 km/h. What is the total length of the route?",
        "answer": "150",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51537,
        "difficulty": null
      },
      {
        "id": "1444427ab7d0a02d746b780e7672f09be123b9afa003fc49061234f1eabc6514",
        "domain": "numina",
        "question": "12. Let $f(x)=x^{2}+6 x+c$ for all real numbers $x$, where $c$ is some real number. For what values of $c$ does $f(f(x))$ have exactly 3 distinct real roots?",
        "answer": "\\frac{11-\\sqrt{13}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151256,
        "difficulty": null
      },
      {
        "id": "41911642e94f8624eb8f0d7c437546e93c3819abe75502c0ab4178b7858c28c5",
        "domain": "numina",
        "question": "15. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $a_{1}=1, a_{n+1}=\\frac{1}{8} a_{n}^{2}+m\\left(n \\in \\mathbf{N}^{*}\\right)$, if for any positive integer $n$, we have $a_{n}<4$, find the maximum value of the real number $m$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91604,
        "difficulty": null
      },
      {
        "id": "d6f03a7e8234378268d84b0fa47a128b89303a401309244c5fb2e1ec80975e3e",
        "domain": "numina",
        "question": "## 268. Math Puzzle $9 / 87$\n\nAt the closing ceremony of a Pioneer camp, a 150-liter soda keg was tapped. Unfortunately, it was a bit cooler than expected that day, so at the end of the event, 20 liters more soda remained in the keg than had been served.\n\nHow many liters did the Pioneers drink?",
        "answer": "65",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181938,
        "difficulty": null
      },
      {
        "id": "d9f3b47c33d3e10d8fcde4f7633f7551c232fdaacfe85f97ae761d6cb2001bec",
        "domain": "numina",
        "question": "1. The minimum value of the function $f(x)=\\sin 2022 x+|x-11|+|x-27|$ is",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129563,
        "difficulty": null
      },
      {
        "id": "321451ce888d12ff8b5ddce316e017989eda98b022dcb26fe8b91b705097c376",
        "domain": "numina",
        "question": "3. (3 points) Wang Qiang bought 6 notebooks and 4 pencils for a total of 9.2 yuan, Zhou Jun bought the same 3 notebooks and 1 pencil, for a total of 3.8 yuan. Then, the cost of buying one notebook and one pencil should be $\\qquad$ yuan.",
        "answer": "1.8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141247,
        "difficulty": null
      },
      {
        "id": "2af384f387d1d8aff1cda37624eb893685c9a4037691ce2367a5562081db9cb1",
        "domain": "numina",
        "question": "1. Fill two $a$s and two $b$s into the 16 cells shown in Figure 3, with at most one letter per cell. If the same letters must not be in the same row or column, find the number of different ways to fill the cells.\n(2007, National High School Mathematics Competition)",
        "answer": "3960",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104285,
        "difficulty": null
      },
      {
        "id": "e1ed484304b50fe3ba53edea8efff8e9ce73bf5f81f152dd9fc4c7ea5d32ac08",
        "domain": "numina",
        "question": "## Task B-2.1.\n\nSolve the equation $\\sqrt{3-\\frac{1}{2020 x}}=1-\\frac{1}{2020 x}$.",
        "answer": "-\\frac{1}{2020}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88783,
        "difficulty": null
      },
      {
        "id": "922336ab67434cedf8a58a1187c90fe7c4c85fc4a411ee1fdb2e00aef38919f2",
        "domain": "numina",
        "question": "243. Christmas Cryptarithm. The Christmas greeting $A M E R R Y X M A S T O A L L^{*}$ is a cryptarithm where each letter represents a specific decimal digit, and each word replaces a square number. Decode this message, given that, in addition, the sum of the digits in each word also matches some square.\n\nTranslate the above text into English, please retain the line breaks and format of the source text, and output the translation result directly.",
        "answer": "934225739681900",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 41676,
        "difficulty": null
      },
      {
        "id": "0be7f6eb863fa932cd1a0b19f4a0c62bed32733e3e5e08716d8122a527f4e6b4",
        "domain": "numina",
        "question": "2. Given the lines $l_{1}: y=x, l_{2}: y=2 x$, then the slope of the line $l$ that forms equal angles with $l_{1}, l_{2}$ is",
        "answer": "\\frac{1\\\\sqrt{10}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124836,
        "difficulty": null
      },
      {
        "id": "fe46d37ebb04f5b40ac56b058227f0ca2f77c026e5a539f93275e242e2320dd7",
        "domain": "numina",
        "question": "12. (6 points) There are 9 steel balls, 8 of which weigh the same, and one is slightly lighter than the other 8. Using a balance scale, what is the minimum number of weighings required to find the lighter steel ball?",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 143105,
        "difficulty": null
      },
      {
        "id": "777744c4921dc0b0ae13759b9f4471412a227bbf4228e06c8113513f82d322a7",
        "domain": "numina",
        "question": "8. (10th \"Hope Cup\" Invitational Competition Question) Let $x_{1}, x_{2}$ be the roots of the equation $x^{2}+x+1=0$, then the value of the sum $\\frac{x_{1}}{x_{2}}+\\left(\\frac{x_{1}}{x_{2}}\\right)^{2}+\\left(\\frac{x_{1}}{x_{2}}\\right)^{3}+\\cdots+\\left(\\frac{x_{1}}{x_{2}}\\right)^{1998}$ is $\\qquad$ $ـ$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95926,
        "difficulty": null
      },
      {
        "id": "dd35ed38d7ca19c3fa826a953fd1e8b607ee575b3d8dc3b03448c06cdea7c451",
        "domain": "numina",
        "question": "2. There is a fraction, when the numerator is reduced by 2021 and the denominator is increased by 2021, it simplifies to $\\frac{1}{2021}$. Then the minimum value of the sum of the original fraction's numerator and denominator is",
        "answer": "4044",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146211,
        "difficulty": null
      },
      {
        "id": "2647d422dbb3a2e0827aa32b427d05ffd1f6e90c90ae3aa67885d35c232706d8",
        "domain": "numina",
        "question": "8. (5 points) A fast train and a slow train are traveling towards each other. The length of the fast train is 315 meters, and the length of the slow train is 300 meters. A person sitting on the slow train sees the fast train pass by in 21 seconds. How many seconds does it take for a person sitting on the fast train to see the slow train pass by?",
        "answer": "20",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 98710,
        "difficulty": null
      },
      {
        "id": "4263367ef23b74dd9bc4f41c8bf68f23e25eedd9459f5e4ad5aea603d3304ca5",
        "domain": "numina",
        "question": "\\section*{Task 1 - V11231}\n\nIn a large state-owned enterprise in the electrical industry, 12,000 units of a certain semi-finished product are annually procured from a supplier at a price of 1.00 DM per unit. The orders were previously placed twice a year, on January 1 and July 1.\n\nThe administrative costs for each order (issuing and sending the order, monitoring the delivery date, invoice review, booking, etc.) amount to 30.00 DM. The storage costs (space costs, administration, \"shrinkage\" due to spoilage and damage, etc.) amount to 20% of the value of the average inventory. The costs for ordering and storage were therefore:\n\n2 orders ... 60.00 DM\n\nStorage costs 20% of the average inventory (3000 units), i.e., 20% of 3000.00 DM, which is 600.00 DM\n\nTotal 660.00 DM\n\nIn a production meeting, it is suggested to reduce costs by ordering the quantity needed for each quarter (3000 units) four times a year.\n\na) How high are the costs according to this suggestion?\n\nb) For how many orders do the lowest costs occur? How high are the costs in this case?\n\nHint: The term \"Deutsche Mark\" was only changed to \"Mark der Deutschen Notenbank\" (MDN) in 1964 in East Germany, and subsequently to \"Mark\" in 1968.",
        "answer": "380",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 185599,
        "difficulty": null
      },
      {
        "id": "ecf8d93dba77cd112ed8dc483f0525286f5dbf0e628ec13db96607d933edf189",
        "domain": "numina",
        "question": "## Problem 3\n\nA tetrahedron has at most one edge longer than 1 . What is the maximum total length of its edges?",
        "answer": "5+\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187153,
        "difficulty": null
      },
      {
        "id": "9ba46cae5fd4a455d485e09c09b623bc726e0dea0d6fe46113af12dcf69e36b1",
        "domain": "numina",
        "question": "A cube is divided by planes that are perpendicular to each of its body diagonals and pass through their midpoints. How many parts does these 4 planes divide the surface of the cube into? - How many parts are created if perpendicular planes are placed at both one-third points of each body diagonal? (8 planes).",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173507,
        "difficulty": null
      },
      {
        "id": "856b292942ccf2c04abe896314bd450c8efa759293b7963377e99cbe251505f8",
        "domain": "numina",
        "question": "2B. On Monday, three bananas cost as much as a lemon and an orange together. On Tuesday, the prices of all fruits were reduced by the same amount of money, two oranges cost as much as three bananas and one lemon, and the price of half a lemon was 5 denars.\n\nWhat was the price of one orange on Monday?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190687,
        "difficulty": null
      },
      {
        "id": "c74b19fbf05b5fd53756a6a42b3c94d9a055e93ea65094efa1f362d24fd71f1a",
        "domain": "numina",
        "question": "A mathematical worm starts from point $A$ and reaches point $B$ in the figure below.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=426&width=436&top_left_y=453&top_left_x=772)\n\nB\n\nThis mathematical worm always moves along the black lines in the drawing above, and never passes over a place it has already been to. Additionally, this worm can move down, up, and to the right, but not to the left. For example, a possible path for the mathematical worm to go from point $A$ to point $B$ could be:\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=420&width=419&top_left_y=1298&top_left_x=790)\n\na) In how many different ways can the mathematical worm go from point $A$ to point $B$ through paths contained in the segments shown in the figure below? (following the rules described above).\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=466&width=440&top_left_y=2020&top_left_x=751)\n\nb) What is the total number of ways the mathematical worm can go from point $A$ to point $B$? (following the previous rules, for any path, not just those in part a)).",
        "answer": "117649",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73860,
        "difficulty": null
      },
      {
        "id": "df6d6d698d98a434884d92315c4660b7108250ba95e5dd5447e538f2a8d174d8",
        "domain": "numina",
        "question": "2. The octahedron $A B C D E F$ has a square $A B C D$ as its base, while the line $E F$ is normal to the plane determined by the square $A B C D$ and passes through its center. It is known that the sphere that touches all faces of the octahedron and the sphere that touches all lateral edges (i.e., $E A, E B, E C, E D, F A, F B, F C$ and $F D$) have the same center, and that the sphere that touches the lateral edges has a surface area that is $50\\%$ larger. Find the ratio $\\frac{E F}{A B}$.",
        "answer": "\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 161108,
        "difficulty": null
      },
      {
        "id": "65be61e7a2b6d9d0a93c1b5c24644a877c15b4b2eaf9f85f6713667be0918248",
        "domain": "numina",
        "question": "3. Grey Wolf needs to cut a 20-meter long steel pipe into 4-meter long segments, and it takes 2 minutes for each cut. How many minutes will it take in total?",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 131043,
        "difficulty": null
      },
      {
        "id": "4fac3ed6b8a43d661ff35c2d5bc5956f7dd92acb85f5767ef80d043ef663d8c9",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{4 n^{2}-\\sqrt[4]{n^{3}}}{\\sqrt[3]{n^{6}+n^{3}+1}-5 n}$",
        "answer": "4",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47476,
        "difficulty": null
      },
      {
        "id": "633c1d4918665ac58eebe38b3b0775afac1b3219b0fec7f507667a00b8194b1d",
        "domain": "numina",
        "question": "Anthroov A.\n\nSeveral powers of two are written in a row on the board in some order. For each pair of adjacent numbers, Petya wrote down in his notebook the power to which the left number must be raised to get the right number. The first number in the row on the board was 2, and the last number was 1024. Vasya claims that this is enough to find the product of all the numbers in the notebook. Is Vasya right?",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68529,
        "difficulty": null
      },
      {
        "id": "cbe93e254ca63fb3950d3a07f9d99f561a93fc124b6042c56220ca6a6fb0a4be",
        "domain": "numina",
        "question": "In the tin, there were red and green candies. Čeněk ate $\\frac{2}{5}$ of all the red candies and Zuzka ate $\\frac{3}{5}$ of all the green candies. Now, the red candies make up $\\frac{3}{8}$ of all the candies in the tin.\n\nHow many candies were there in the tin at least originally?\n\n(L. Růžičková)\n\nHint. How many candies of each color could, or could not, have been originally in the tin?",
        "answer": "35",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76801,
        "difficulty": null
      },
      {
        "id": "cc3dcf9a89d144f752a2b939579c0806dd364bfb3dd7af9c00a71d3a3af0dd2f",
        "domain": "numina",
        "question": "9. $[7]$ Let $A B C$ be a triangle, and $I$ its incenter. Let the incircle of $A B C$ touch side $B C$ at $D$, and let lines $B I$ and $C I$ meet the circle with diameter $A I$ at points $P$ and $Q$, respectively. Given $B I=$ $6, C I=5, D I=3$, determine the value of $(D P / D Q)^{2}$.",
        "answer": "\\frac{75}{64}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150955,
        "difficulty": null
      },
      {
        "id": "474385042f396f36e4b9cf625af3c7676dc98a5f736ec901189d401e415509a6",
        "domain": "numina",
        "question": "5. How many five-digit natural numbers are there in which all digits are different, and among which the digits 1 and 2 appear in adjacent decimal places?\n\nThe use of a pocket calculator or any reference materials is not allowed.\n\n# Ministry of Science and Education of the Republic of Croatia Agency for Education and Education Croatian Mathematical Society \n\nCOUNTY COMPETITION IN MATHEMATICS February 26, 2024. 6th grade - elementary school\n\nEach task is worth 10 points. In addition to the final result, the process is also graded. To earn all points, it is necessary to find all solutions, determine that there are no others, document the process, and justify your conclusions.",
        "answer": "2436",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86493,
        "difficulty": null
      },
      {
        "id": "6da1e0fafa48e193afd8bef60e464560ba4e356bb85eb4aa112e44f9e579a955",
        "domain": "numina",
        "question": "A circle is inscribed in a right angle. A chord connecting the points of tangency is equal to 2. Find the distance from the center of the circle to this chord.\n\n#",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57960,
        "difficulty": null
      },
      {
        "id": "1f74d68f31940a437882434b16765aa57b6b83aa9067dcebca689c5e07469327",
        "domain": "numina",
        "question": "1. Determine the number of distinct values of the expression\n\n$$\n\\frac{n^{2}-2}{n^{2}-n+2}\n$$\n\nwhere $n \\in\\{1,2, \\ldots, 100\\}$.",
        "answer": "98",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86321,
        "difficulty": null
      },
      {
        "id": "bdc7e5e040753021094f46a6498f56456edcbae1f802cfb6ae2d845ddbd42dbd",
        "domain": "numina",
        "question": "The 83 interns from the Animath internship each choose an activity for the free afternoon from 5 proposed activities. We know that:\n\n$\\triangleright$ Shopping has been at least as popular as Laser Game;\n\n$\\Delta$ Movie tickets are sold in lots of 6;\n\n$\\Delta$ At most 5 students go to the beach;\n\n$\\triangleright$ At most one student has visited a Romanesque church.\n\nWhen writing the list of activities in alphabetical order, one writes the number of students corresponding to each activity. How many different lists of numbers could have been written?",
        "answer": "3570",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 82475,
        "difficulty": null
      },
      {
        "id": "7a0c0f812cf961437881d8696bf4e014a8501686bebea827a856bdcd9ca37bed",
        "domain": "numina",
        "question": "14. (POL 2) For any positive integer $x$ define  $$ \\begin{aligned} g(x) & =\\text { greatest odd divisor of } x, \\\\ f(x) & =\\left\\{\\begin{array}{l} x / 2+x / g(x), \\text { if } x \\text { is even; } \\\\ 2^{(x+1) / 2}, \\text { if } x \\text { is odd. } \\end{array}\\right. \\end{aligned} $$  Construct the sequence $x_{1}=1, x_{n+1}=f\\left(x_{n}\\right)$. Show that the number 1992 appears in this sequence, determine the least $n$ such that $x_{n}=1992$, and determine whether $n$ is unique.",
        "answer": "8253",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 197057,
        "difficulty": null
      },
      {
        "id": "16e28b17a2ea5bd2f34ab5466b5f168855b03bc3a0cd80f63cbc0be33e30ce4a",
        "domain": "numina",
        "question": "13.022. On the mathematics entrance exam, $15 \\%$ of the applicants did not solve a single problem, 144 people solved problems with errors, and the number of those who solved all problems correctly is to the number of those who did not solve any at all as 5:3. How many people took the mathematics exam that day",
        "answer": "240",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51530,
        "difficulty": null
      },
      {
        "id": "4c06a18d190001344f8fba930c7525054d9abcd646887f955fb5b932f2ecb76a",
        "domain": "numina",
        "question": "3. The price of one pencil is a whole number of denars. The price of 9 such pencils is greater than 1100 denars, but less than 1200 denars. The price of 13 pencils is greater than 1500 denars, but less than 1600 denars. How much does one pencil cost?",
        "answer": "123",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 189399,
        "difficulty": null
      },
      {
        "id": "4b4f5f3dfbafb8e864209f11d6b72ab9f8ea179da7fb6a804b35861aa559f884",
        "domain": "numina",
        "question": "9. A9 (POL) Let the sequence \\(a(n), n=1,2,3, \\ldots\\), be generated as follows: \\(a(1)=0\\), and for \\(n>1\\), \n\\[ a(n)=a([n / 2])+(-1)^{\\frac{n(n+1)}{2}} . \\quad(\\text{Here }[t]=\\text{ the greatest integer } \\leq t .) \\]\n(a) Determine the maximum and minimum value of \\(a(n)\\) over \\(n \\leq 1996\\) and find all \\(n \\leq 1996\\) for which these extreme values are attained.\n(b) How many terms \\(a(n), n \\leq 1996\\), are equal to 0?",
        "answer": "346",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 197165,
        "difficulty": null
      },
      {
        "id": "56456e46fb7d5fc897307865073070ce8668fc9d7a9e399c76b5ef4c5a84a630",
        "domain": "numina",
        "question": "12. (10 points) There are four colors of light bulbs (with an ample supply of each color), and one light bulb needs to be installed at each vertex of the triangular prism $A B C-A_{1} B_{1} C_{1}$. The requirement is that the light bulbs at the two endpoints of the same edge must be of different colors, and each color of light bulb must be used at least once. How many installation methods are there?",
        "answer": "216",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132248,
        "difficulty": null
      },
      {
        "id": "0b3296c1dcdb9a79c70ba61ffd1a799b25b247a7041bf63a359e315b68b70a8b",
        "domain": "numina",
        "question": "## Task B-3.6.\n\nInside the square $A B C D$ there is a point $M$ such that $|M A|=7 \\mathrm{~cm}, |M B|=13 \\mathrm{~cm}$ and $|M C|=17 \\mathrm{~cm}$. Calculate the area of the square $A B C D$.",
        "answer": "288",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89348,
        "difficulty": null
      },
      {
        "id": "9e8ab03da1fa39bab73135bd9c380d5cfb2cf3ef020009f48338d2c2d61d4a12",
        "domain": "numina",
        "question": "4. If the positive integer $n$ makes $\\sqrt{3}$ always lie between $1+\\frac{3}{n}$ and $1+\\frac{3}{n+1}$, then $n=$",
        "answer": "4",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 129969,
        "difficulty": null
      },
      {
        "id": "292b4fc23e2eaa8b7088556dde6474ca998f2cc59d9e193ec5d54ec0e9dda28c",
        "domain": "numina",
        "question": "11. Use 100 RMB to buy items, with the rule that you can only buy items in one of the following three ways each day: (1) buy candy for 1 RMB; (2) buy cookies for 2 RMB; (3) buy fruit for 2 RMB, and you are not allowed to buy nothing. There are $\\qquad$ ways to spend all 100 RMB.",
        "answer": "\\frac{1}{3}(2^{101}+1)",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 97424,
        "difficulty": null
      },
      {
        "id": "a4c2979981288d2933d189596b472af49d0108fc0eab9643e6903091c77c112f",
        "domain": "numina",
        "question": "8. [30] Consider the following two-player game. Player 1 starts with a number, $N$. He then subtracts a proper divisor of $N$ from $N$ and gives the result to player 2 (a proper divisor of $N$ is a positive divisor of $N$ that is not equal to 1 or $N$ ). Player 2 does the same thing with the number she gets from player 1 , and gives the result back to player 1. The two players continue until a player is given a prime number, at which point that player loses. For how many values of $N$ between 2 and 100 inclusive does player 1 have a winning strategy?",
        "answer": "47",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151227,
        "difficulty": null
      },
      {
        "id": "8edbb500206c6c7f241b29dc7746340d861346f09d921bbb25404fc407ae292e",
        "domain": "numina",
        "question": "2. Let $P$ be the intersection of the diagonals of a convex quadrilateral $A B C D$. Calculate its area if the areas of triangles $A B C, B C D$ and $D A P$ are $8 \\mathrm{~cm}^{2}, 9 \\mathrm{~cm}^{2}$, $10 \\mathrm{~cm}^{2}$, respectively.",
        "answer": "24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75949,
        "difficulty": null
      },
      {
        "id": "306cf2996c4e6e175bc967e2b2c91264bd46b1db2a313ef664536e97a92618a3",
        "domain": "numina",
        "question": ". Let $ABCD$ be an isosceles trapezoid, with the larger base $[AB]$. Suppose the diagonals intersect at a point $O$ such that $\\frac{OA}{OC}=2$. Given that the area of triangle $BOC$ is 10, what is the area of trapezoid $ABCD$?",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 81449,
        "difficulty": null
      },
      {
        "id": "e4959bc4a4944bcc3c8b3682f58bc4cb5a1488fe20a1ea24c76c26e7671d6852",
        "domain": "numina",
        "question": "2. (2 points) Find the minimum value of the expression $4 x^{2}+4 x \\sin y-\\cos ^{2} y$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 20524,
        "difficulty": null
      },
      {
        "id": "46aad0776bc3f17d8b4e60ef2f520b0a73c5a2489a823adb42d14ec7e0dd76b0",
        "domain": "numina",
        "question": "I3.4 $x, y$ are real numbers. If $x+y=c-195$ and $d$ is the maximum value of $x y$, find $d$.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152519,
        "difficulty": null
      },
      {
        "id": "f4fb8032195b2a4cabb3b78014f4fc836c453db341d0a216a85291ae6b056e94",
        "domain": "numina",
        "question": "## Task 6B - 171246B\n\na) Given a natural number $n \\geq 2$. Let $u$ be the circumcircle of a regular $2 n$-gon $A_{0} A_{1} \\ldots A_{2 n-1}$.\n\nA set of three vertices $A_{i}, A_{j}, A_{k}$ of this $2 n$-gon is called one-sided if there is a semicircular arc $h$ on the circumference $u$, including its two endpoints, that contains $A_{i}, A_{j}$, and $A_{k}$.\n\nDetermine the probability $w$ that a randomly chosen set $M=\\left\\{A_{i}, A_{j}, A_{k}\\right\\}$ of three vertices is one-sided.\n\nb) Determine the limit $\\lim _{n \\rightarrow \\infty} w_{n}$ if it exists.\n\nHint:\n\nIf $m_{n}$ is the number of all sets that can be formed from three vertices $A_{i}, A_{j}, A_{k}$ of the $2 n$-gon, and $g_{n}$ is the number of one-sided sets among them, then the probability sought in a) is defined as $w_{n}=\\frac{g_{n}}{m_{n}}$.",
        "answer": "\\frac{3}{4}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 185945,
        "difficulty": null
      },
      {
        "id": "a9119c1d0dd1fe28f369a70493d162d5166f33fd6a4771306858486d967978de",
        "domain": "numina",
        "question": "Example 13 Find the minimum value of the bivariate function $f(x, y)=(x-y)^{2}+\\left(x+\\frac{1}{y}+1\\right)^{2}$.\n(1998 \"Hope Cup\" Competition Question)",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145529,
        "difficulty": null
      },
      {
        "id": "6a9d7dadef5ef1881a68a58b26d78c243571ce8903df41634d13ccbbf24220cb",
        "domain": "numina",
        "question": "45a * If positive numbers $a, b, c$ and constant $k$ satisfy the inequality $\\frac{k a b c}{a+b+c} \\leqslant(a+b)^{2}+(a+b+$ $4 c)^{2}$, find the maximum value of the constant $k$.",
        "answer": "100",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 104962,
        "difficulty": null
      },
      {
        "id": "11815a306283d36cf44869922193e81fd432a0fdf2cc13ce9fa79c0bb5614c52",
        "domain": "numina",
        "question": "$4 \\cdot 218$ In a middle school mathematics competition, three problems $A$, $B$, and $C$ were given. Among the 25 students who participated in the competition, each student solved at least one problem. Among those who did not solve $A$, the number of students who solved $B$ is twice the number of students who solved $C$. The number of students who solved only $A$ is one more than the number of students who solved $A$ among the remaining students. Among the students who solved only one problem, half did not solve $A$. How many students solved only $B$?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 108566,
        "difficulty": null
      },
      {
        "id": "14826f8a9350cae0bb64cfdb1da382c9941f5df37f55f083cbc54a760630f757",
        "domain": "numina",
        "question": "68. For the red clover variety Perm local, on average $84\\%$ of the plants are late maturing. What is the probability that 52 plants out of 60 clover plants, randomly selected, will be late maturing?",
        "answer": "0.1201",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 31691,
        "difficulty": null
      },
      {
        "id": "d62f08aff9af72381c1c1dfd3a2e162ba58969e3b568e424546e20087b2c0f0d",
        "domain": "numina",
        "question": "For which values of $m \\in \\mathbb{R}$ does the equation:\n\n$$\n\\left(x^{2}-2 m x-4\\left(m^{2}+1\\right)\\right)\\left(x^{2}-4 x-2 m\\left(m^{2}+1\\right)\\right)=0\n$$\n\nhave exactly three distinct roots?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 83692,
        "difficulty": null
      },
      {
        "id": "4733a3b4061a5a430b4943aeedd407c35b61ec0c3756beb0bff41c0c57ce4e71",
        "domain": "numina",
        "question": "3. The diagonals of the inscribed quadrilateral $ABCD$ intersect at point $O$. Inside triangle $AOB$, a point $K$ is chosen such that line $KO$ is the bisector of angle $CK$. Ray $DK$ intersects the circumcircle of triangle $COK$ again at point $L$, and ray $CK$ intersects the circumcircle of triangle $DOK$ again at point $M$. Find the ratio of the areas of triangles $ALO$ and $BMO$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7992,
        "difficulty": null
      },
      {
        "id": "61587dc755447a03e5361547561e0512c2ade77a6ea24ae7fd83a3dd48e598e2",
        "domain": "numina",
        "question": "3.1. A number was multiplied by its first digit and the result was 494, by its second digit - 988, by its third digit - 1729. Find this number.",
        "answer": "247",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13744,
        "difficulty": null
      },
      {
        "id": "f05bb9d7897b8429e398f2e4b332c5c2a3ba4b4e359c0e028a6b8ecb572cf8b7",
        "domain": "numina",
        "question": "25. Form a five-digit number without repeating digits using $1,3,5,7,9$, and form another five-digit number without repeating digits using $0,2,4,6,8$. Subtract the smaller number from the larger number among these two five-digit numbers, the smallest possible difference is $\\qquad$",
        "answer": "517",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96860,
        "difficulty": null
      },
      {
        "id": "534341c7c5b865881922f6452ba839d89f5001a09cf0313920cdce11a11736e6",
        "domain": "numina",
        "question": "4. For a finite arithmetic sequence $\\left\\{a_{n}\\right\\}$ with the first term $a_{1}=1$ and a common difference of 2, the arithmetic mean of all its terms is 2015. If one term is deleted, the arithmetic mean of the remaining terms is an integer. Then the number of ways to delete a term is $\\qquad$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120555,
        "difficulty": null
      },
      {
        "id": "0df54735bbeefe34468a5a3ca5cbaf8e21effe23daa231bd3185bfd25088e5d5",
        "domain": "numina",
        "question": "1. Let the numbers be $m=(1+2+3)^{4}+5 \\cdot 6 \\cdot(7+8+9)$ and $n=1+2+3+\\ldots+63$.\n\na) (3p) Calculate $2016^{m-n}$.\n\nb) (4p) Determine the largest and the smallest non-zero natural number which, when divided by $n$, gives a quotient equal to a quarter of the remainder.\n\nProf. Stela Boghian",
        "answer": "10160602020",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 179226,
        "difficulty": null
      },
      {
        "id": "7012db71622f8764772f4bceda3e4af722f5677a3c02c0177d38c4ac3c7713a3",
        "domain": "numina",
        "question": "Folkpor\n\nFive identical balls are moving in one direction in a straight line at some distance from each other, and five other identical balls are moving towards them. The speeds of all the balls are the same. When any two balls collide, they fly apart in opposite directions with the same speed they had before the collision. How many total collisions will occur between the balls?\n\n#",
        "answer": "25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 72179,
        "difficulty": null
      },
      {
        "id": "83148fe6b3a49310aaaa3cb41cd182a6bbc7e6f16a4115ead2a117f7ee8e88a9",
        "domain": "numina",
        "question": "8. (10 points) Through a point $P$ inside an equilateral triangle $A B C$, perpendiculars are drawn to the three sides, with the feet of the perpendiculars being $D, E, F$ respectively. Connect $A P$, $B P$, $C P$. If the area of the equilateral triangle $A B C$ is 2028 square centimeters, and the areas of triangles $P A D$ and $P B E$ are both 192 square centimeters, then the area of triangle $P C F$ is $\\qquad$ square centimeters.",
        "answer": "630",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129765,
        "difficulty": null
      },
      {
        "id": "e0b08339edc095082a4cdc5259119c39a61a2f453617ee2c0f36a212c0133826",
        "domain": "numina",
        "question": "Podolsky A.\n\nChichikov is playing with Nozdryov. First, Nozdryov distributes 1001 nuts among three small boxes. After looking at the distribution, Chichikov names any integer \\( N \\) from 1 to 1001. Then Nozdryov must, if necessary, move one or several nuts to an empty fourth box and show Chichikov one or several boxes where the total number of nuts is exactly \\( N \\). As a result, Chichikov will receive as many dead souls as the number of nuts Nozdryov moved. What is the maximum number of souls Chichikov can guarantee for himself, regardless of how Nozdryov plays?",
        "answer": "71",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53676,
        "difficulty": null
      },
      {
        "id": "dd59656609b72325fba8548593c918e0c2ec4b6131fb56142ca532f046859fbb",
        "domain": "numina",
        "question": "1.58. Perpendiculars are drawn from the vertex of the acute angle of a rhombus to the lines containing the sides of the rhombus to which this vertex does not belong. The length of each perpendicular is 3 cm, and the distance between their bases is $3 \\sqrt{3}$ cm. Calculate the lengths of the diagonals of the rhombus.",
        "answer": "2\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 50087,
        "difficulty": null
      },
      {
        "id": "ce961d8ca00b9ef8a2a7b10bd72939640d133abb46f41f22b8cb98d39cfe006e",
        "domain": "numina",
        "question": "3. Determine the number of all triples of distinct three-digit natural numbers, the sum of which is divisible by each of the three addends.",
        "answer": "234",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75225,
        "difficulty": null
      },
      {
        "id": "2d8794da268ccaaf0f1acc7080574c76b32f33094bc47bb22a80ab47fba623aa",
        "domain": "numina",
        "question": "29. [16] Let $A B C$ be a triangle such that $A B=A C=182$ and $B C=140$. Let $X_{1}$ lie on $A C$ such that $C X_{1}=130$. Let the line through $X_{1}$ perpendicular to $B X_{1}$ at $X_{1}$ meet $A B$ at $X_{2}$. Define $X_{2}, X_{3}, \\ldots$, as follows: for $n$ odd and $n \\geq 1$, let $X_{n+1}$ be the intersection of $A B$ with the perpendicular to $X_{n-1} X_{n}$ through $X_{n}$; for $n$ even and $n \\geq 2$, let $X_{n+1}$ be the intersection of $A C$ with the perpendicular to $X_{n-1} X_{n}$ through $X_{n}$. Find $B X_{1}+X_{1} X_{2}+X_{2} X_{3}+\\ldots$",
        "answer": "\\frac{1106}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151348,
        "difficulty": null
      },
      {
        "id": "3d835b5987674316b52bc7d4f3dd3f105a1a7bfa184480c0907b431fdf3b943f",
        "domain": "numina",
        "question": "An arithmetic sequence has a common difference, $d$, that is a positive integer and is greater than 1. The sequence includes the three terms 3, 468 and 2018. What is the sum of all of the possible values of $d$ ?\n\n(An arithmetic sequence is a sequence in which each term after the first is obtained from the previous term by adding a constant, called the common difference. For example, $3,5,7,9$ are the first four terms of an arithmetic sequence with common difference 2.)",
        "answer": "191",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194036,
        "difficulty": null
      },
      {
        "id": "7c01122a94f13f017410500e2e8db4221c90996d374b546b85985fedd54aac10",
        "domain": "numina",
        "question": "4. Given a parallelogram $A B C D$. A line passing through point $D$ intersects segment $A C$ at point $G$, segment $B C$ at point $F$, and ray $A B$ at point $E$ such that triangles $BEF$ and $CGF$ have the same area. Determine the ratio $|A G|:|G C|$.",
        "answer": "\\frac{1}{2}(1+\\sqrt{5})",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75372,
        "difficulty": null
      },
      {
        "id": "bb6d43c2cef686bed3c299bf86d16128f8911f230ae178424552d59db5a11150",
        "domain": "numina",
        "question": "3-5. Determine the maximum value of the ratio of a three-digit number to the number equal to the sum of the digits of this number.",
        "answer": "100",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 26392,
        "difficulty": null
      },
      {
        "id": "8b57ff084fb247587af9600c7ecfac443203f87636039b80f1c57b8fc6a6b359",
        "domain": "numina",
        "question": "11.1. Does there exist a natural number $n$, greater than 1, such that the value of the expression $\\sqrt{n \\sqrt{n \\sqrt{n}}}$ is a natural number?",
        "answer": "128",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 18268,
        "difficulty": null
      },
      {
        "id": "a0f2cc996e1da2bfdf90b6ae2851c94a16423580b221bbcc01d36ca8140459ca",
        "domain": "numina",
        "question": "B1. Determine the smallest possible value of the expression $\\left|25^{m}-36^{n}\\right|$, where $m$ and $n$ are natural numbers.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 163037,
        "difficulty": null
      },
      {
        "id": "f0b1bb4fa9087c5443683dcc0c484d02b7258fb240240a6c3c9c9d231aee02a8",
        "domain": "numina",
        "question": "14. Let $[x]$ denote the greatest integer not exceeding $x$, e.g. $[\\pi]=3,[5.31]=5$ and $[2010]=2010$. Given $f(0)=0$ and $f(n)=f\\left(\\left[\\frac{n}{2}\\right]\\right)+n-2\\left[\\frac{n}{2}\\right]$ for any positive integer $n$. If $m$ is a positive integer not exceeding 2010, find the greatest possible value of $f(m)$.\n(2 marks)\n設 $[x]$ 表示不超過 $x$ 的最大整數, 例如： $[\\pi]=3 、[5.31]=5 、[2010]=2010$ 。已知 $f(0)=0$, 且對任意正整數 $n$ 皆有 $f(n)=f\\left(\\left[\\frac{n}{2}\\right]\\right)+n-2\\left[\\frac{n}{2}\\right]$ 。若 $m$ 是不超過 2010 的正整數, 求 $f(m)$ 的最大可能值。\n(2 分)",
        "answer": "10",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153606,
        "difficulty": null
      },
      {
        "id": "fb3f5f1579381a467db7d6e3584317b64e334a0179a047566d85446c4c733aad",
        "domain": "numina",
        "question": "Bororin P.A.\n\nOn the surface of a spherical planet, there are four continents separated from each other by an ocean. We will call a point in the ocean special if for it there are at least three nearest (equidistant from it) land points, and all are on different continents. What is the maximum number of special points that can exist on this planet?",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65110,
        "difficulty": null
      },
      {
        "id": "dab9b6453c9eaed68d12d0745d3900cebc6faba8e996ff17fc5f821e3052585b",
        "domain": "numina",
        "question": "Petrov booked an apartment in a new building with five identical entrances. Initially, the entrances were numbered from left to right, and Petrov's apartment had the number 636. Then the developer changed the numbering to the opposite direction (from right to left, see the figure). As a result, Petrov's apartment became number 242. How many apartments are in the building? (The order of apartment numbering within each entrance did not change.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d3cd76ad0dc53ae543bdg-43.jpg?height=89&width=734&top_left_y=184&top_left_x=662)",
        "answer": "985",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 66741,
        "difficulty": null
      },
      {
        "id": "c77bf64434bd6a460f5b231829a488359fe5401a3eb5c658575e41ab45d5bd96",
        "domain": "numina",
        "question": "8.5. Each digit of the natural number $N$ is strictly greater than the one to its left. What is the sum of the digits of the number $9 N$?",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9482,
        "difficulty": null
      },
      {
        "id": "f3e3381c56d17f902fcade0eeed8b17acebb94793ccb0c456eef9ace2fe3491a",
        "domain": "numina",
        "question": "## Task 1 - 310711\n\nA department store received a shipment of red, blue, and green balls, totaling 675 pieces. During a certain period, the following were sold:\n\nhalf of the red balls, two-thirds of the blue balls, and one-quarter of the green balls.\n\nIt turned out that after this, the same number of balls remained for each of the three colors.\n\nDetermine from these data,\n\na) how many balls of each of the three colors were sold during this time.\n\nb) how many balls were still available in total afterward!",
        "answer": "225",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 183835,
        "difficulty": null
      },
      {
        "id": "b988638e429957abe179ae65235a9aff51641d5dc081eb568680d1b5d15985b2",
        "domain": "numina",
        "question": "10. Xiaocuxin made a mistake when calculating the division, writing the dividend 2002 as 2020, resulting in a quotient of 106 and a remainder of 6. The correct quotient should be $\\qquad$ .",
        "answer": "105",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 142633,
        "difficulty": null
      },
      {
        "id": "47a3b203c0fb57777d6ec498493ef13f8232334befeac125491222f001675937",
        "domain": "numina",
        "question": "8,9 [ Inscribed Circle in an Angle\n\nIn an isosceles trapezoid, there are two circles. One of them, with a radius of 1, is inscribed in the trapezoid, and the second circle touches two sides of the trapezoid and the first circle. The distance from the vertex of the angle formed by the two sides of the trapezoid that touch the second circle to the point of tangency of the circles is twice the diameter of the second circle. Find the area of the trapezoid.\n\n#",
        "answer": "\\frac{9\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58538,
        "difficulty": null
      },
      {
        "id": "9b88b628dafcbffa0b4cbf5e3d62b88b910b7f6c1a027e48483308dcb320ae46",
        "domain": "numina",
        "question": "In a right-angled triangle, $s_{a}$ and $s_{b}$ are the medians to the legs, and $s_{c}$ is the median to the hypotenuse. Determine the maximum value of the expression $\\frac{s_{a}+s_{b}}{s_{c}}$.",
        "answer": "\\sqrt{10}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 171209,
        "difficulty": null
      },
      {
        "id": "87b2f868c222e4a08f335ffc17ba47582ed481f828cc7dcfeccdf3306ae7b83c",
        "domain": "numina",
        "question": "On sides $A B$ and $B C$ of an equilateral triangle $A B C$, points $D$ and $K$ are marked, respectively, and on side $A C$, points $E$ and $M$ are marked such that $D A + A E = K C + C M = A B$. Segments $D M$ and $K E$ intersect. Find the angle between them.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62049,
        "difficulty": null
      },
      {
        "id": "f7dbaf6dc7799b49778ed3772b956e311133f7261cec5fbd6e0c61ede6b9d2df",
        "domain": "numina",
        "question": "6. Among all natural numbers from 1 to 20 inclusive, some 10 numbers were painted blue, and the other 10 - red, then all possible sums of pairs of numbers, one of which is blue and the other is red, were counted. What is the maximum number of different sums that can be among the hundred obtained numbers?",
        "answer": "35",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9813,
        "difficulty": null
      },
      {
        "id": "5f8983dfe32d3cbd385b96ef7b5b6426316b46ee872929b3a9c79d8f44eb3a20",
        "domain": "numina",
        "question": "7.3. In the notebook, all irreducible fractions with the numerator 15 are written down, which are greater than $\\frac{1}{16}$ and less than $\\frac{1}{15}$. How many such fractions are written down in the notebook?",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17020,
        "difficulty": null
      },
      {
        "id": "cc791fefbd16763b86f212d2c57b4e8a9706e2d7d3459971367c2cad9557635e",
        "domain": "numina",
        "question": "9. Si Si wrote $1, 2, 3, \\cdots \\cdots, 99$ on the blackboard. Rui Rui doesn't like the number 2, so he erased the following numbers:\n(1) multiples of 2;\n(2) numbers containing the digit 2 (such as 92);\n(3) numbers whose sum of digits is divisible by 2 (such as 31).\nRui Rui erased $\\qquad$ numbers in total.",
        "answer": "79",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113195,
        "difficulty": null
      },
      {
        "id": "2835321a5826e0d222caf12b8d46790099f9f57bc4392bc3b5793b237b47a644",
        "domain": "numina",
        "question": "5. On one side of the road, some power poles are arranged at equal distances. Xiao Ming rides his bike along the road. He took 3 minutes to ride from the 1st power pole to the 10th power pole. At this speed, in another 3 minutes, Xiao Ming can ride to the $\\qquad$th power pole.",
        "answer": "19",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92941,
        "difficulty": null
      },
      {
        "id": "9d775ae34320b0e63317362035dbe5b9ce2473f366e7e2d8f2b0a4205cd4cc80",
        "domain": "numina",
        "question": "7. Use 4 colors to color a $4 \\times 4$ grid, requiring that the 4 cells in the same row and the 4 cells in the same column have different colors. There are $\\qquad$ different coloring methods.",
        "answer": "576",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 128029,
        "difficulty": null
      },
      {
        "id": "03fad349015c9acb8b8503749d230c40658121c89adc03e7ca4f59213aa95f9a",
        "domain": "numina",
        "question": "Problem 5.8. Inside a large triangle with a perimeter of 120, several segments were drawn, dividing it into nine smaller triangles, as shown in the figure. It turned out that the perimeters of all nine small triangles are equal to each other. What can they be equal to? List all possible options.\n\nThe perimeter of a figure is the sum of the lengths of all its sides.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-12.jpg?height=256&width=323&top_left_y=1257&top_left_x=565)",
        "answer": "40",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17062,
        "difficulty": null
      },
      {
        "id": "eb102567f1cd88fa4a6dcfed2bff278cd698c9ed4b94d1a29f549d98e3319bb0",
        "domain": "numina",
        "question": "2. In triangle $ABC$, the lengths of the sides are known: $\\overline{AB}=4, \\overline{BC}=7$ and $\\overline{AC}=5$. Let $\\alpha=\\angle BAC$. Calculate the value of the expression $\\sin ^{6} \\frac{\\alpha}{2}+\\cos ^{6} \\frac{\\alpha}{2}$.",
        "answer": "\\frac{7}{25}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187369,
        "difficulty": null
      },
      {
        "id": "8ceb6f247092cd65a50208c50dcfe0d106e982fe76b519f61da8b7c80f4471a6",
        "domain": "numina",
        "question": "8.68. Using a compass and a ruler, divide an angle of $19^{\\circ}$ into 19 equal parts.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 27287,
        "difficulty": null
      },
      {
        "id": "6ab4f100765fcd2b9ec509843c3b5631ad0c5117730ee0f607f3395afe18d9f1",
        "domain": "numina",
        "question": "Task 1. Dad and Mom are cooking pancakes on the stove, while their children Petya and Vasya are eating them. Dad can cook 70 pancakes in an hour, and Mom can cook 100 pancakes. Petya, if he tries hard, can eat 10 pancakes in 15 minutes, and Vasya can eat twice as much. After how much time will there be no less than 20 uneaten pancakes left on the table?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 2801,
        "difficulty": null
      },
      {
        "id": "08cc1ef46a8c952e79b5ce47c740e7293b2f2380484279ea6e99d19602899f28",
        "domain": "numina",
        "question": "5. In the Cartesian coordinate system, with the point $(1,0)$ as the center and $r$ as the radius, a circle intersects the parabola $y^{2}=x$ at four points $A$, $B, C, D$. If the intersection point $F$ of $A C$ and $B D$ is exactly the focus of the parabola, then $r=$ $\\qquad$ .",
        "answer": "\\frac{\\sqrt{15}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147210,
        "difficulty": null
      },
      {
        "id": "fcd388e78d182ffbc78feeb1be6a2b3c6f2eaa58e196d6fc0cd02df28773e719",
        "domain": "numina",
        "question": "15. Find the sum: $1+2+3+\\ldots+1999$.",
        "answer": "1999000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35774,
        "difficulty": null
      },
      {
        "id": "df07a2531c9b809a805f3740eb19ed442904cfd2502bc51a3113d017f717c66d",
        "domain": "numina",
        "question": "6. Xiao Ming plans to memorize 40 English words from Monday to Sunday over 7 days. He requires himself to memorize a few words each day, with a different number of words memorized each day, and the most words are to be memorized on Sunday. How many words can Xiao Ming memorize at most on Sunday? $\\qquad$",
        "answer": "19",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 111650,
        "difficulty": null
      },
      {
        "id": "5ad702ae53fc20ee179e879d4b11ece4d1268ab80768ac668357f3271acdfdb8",
        "domain": "numina",
        "question": "## Subject II\n\na) Write as an irreducible fraction the sum: $S=\\frac{1}{1+2}+\\frac{1}{1+2+3}+\\ldots+\\frac{1}{1+2+\\cdots+2015}$\n\nb) Show that: $\\frac{1}{2} \\cdot\\left(\\frac{1}{2}+\\frac{1}{2+4}\\right)\\left(\\frac{1}{2}+\\frac{1}{2+4}+\\frac{1}{2+4+6}\\right) \\cdot \\ldots \\cdot\\left(\\frac{1}{2}+\\frac{1}{2+4}+\\cdots+\\frac{1}{2+4+6+\\cdots+4028}\\right)=\\frac{1}{2015}$",
        "answer": "\\frac{1}{2015}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178066,
        "difficulty": null
      },
      {
        "id": "0d7eec39b0882e8ccd99e446e3691ddb58877278ee5591fba83300bc64c38468",
        "domain": "numina",
        "question": "3. (5 points) Think of a number, add 5 to it, multiply the result by 2, subtract 4 from the product, divide the difference by 2, and then subtract the number you initially thought of. The final calculation result is $\\qquad$ .",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128890,
        "difficulty": null
      },
      {
        "id": "f616e42d85befdede3f0d059d11c2ccf9c083823a25ec0a95aa5e754dd5768cb",
        "domain": "numina",
        "question": "2. $a \\in \\mathbf{R}$, the complex number $\\omega=1+a \\mathrm{i}$, and the complex number $z$ satisfies $\\bar{\\omega} z-\\omega=0$. For what value of $a$ does $|z^{2}-z+2|$ have the minimum value, and find this minimum value.",
        "answer": "\\frac{\\sqrt{14}}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129332,
        "difficulty": null
      },
      {
        "id": "54a09ab908c2e4ffe808c71a50c2dee47e4ef5c4479f09bb61f7e82b7944085b",
        "domain": "numina",
        "question": "Problem 5. (6 points) Peter has $n^{3}$ unit white cubes, from which he wants to assemble a cube $n \\times n \\times n$ with a completely white surface. How many of the faces of the unit cubes does Emil need to color to prevent Peter? Solve the problem for:\na) (2 points) $n=2$;\nb) (4 points) $n=3$.",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 177128,
        "difficulty": null
      },
      {
        "id": "2618802c0ec5542cd96a08a4a33b1360b0429e930c28d9f047a5caaa4f323131",
        "domain": "numina",
        "question": "$2.19 \\frac{\\sqrt{x}+\\sqrt[3]{x}}{\\sqrt{x}-\\sqrt[3]{x}}=3$.",
        "answer": "64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49512,
        "difficulty": null
      },
      {
        "id": "59a32ad559a20fac9b41305189a9309140318d76713ab0653f01674d3a545abf",
        "domain": "numina",
        "question": "Calculate $\\sum_{i=1}^{n} \\prod_{j \\neq i} \\frac{x-a_{j}}{a_{i}-a_{j}}$ for $a_{1}, \\ldots, a_{n} \\in \\mathbb{R}$ pairwise distinct.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 81081,
        "difficulty": null
      },
      {
        "id": "aade6e3d2f162487bb40fba80f18e74d4265820d66cf82c6fc02a5ade8ddbb00",
        "domain": "numina",
        "question": "## Task 2.\n\nA natural number $n$ is good if we can color each side and diagonal of a regular $n$-gon in some color so that for any two vertices $A$ and $B$, there is exactly one vertex $C$, different from $A$ and $B$, such that the segments $\\overline{A B}, \\overline{B C}$, and $\\overline{C A}$ are colored the same color.\n\nDetermine which of the numbers $7,8,9,10,11$, and 12 are good, and which are not.",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 90068,
        "difficulty": null
      },
      {
        "id": "b4b963338c0d945e77dd2ded6205014c0ab2a94177d3c88ba65fbd49baab92c5",
        "domain": "numina",
        "question": "Let $P(X)$ be a monic polynomial of degree 2017 such that $P(1)=1, P(2)=2, \\ldots, P(2017)=$ 2017. What is the value of $P(2018)$?",
        "answer": "2017!+2018",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 79153,
        "difficulty": null
      },
      {
        "id": "7e4c087715e308578c5fb983e4bc30e6ce60171da0a90ddb79af615dbc221e60",
        "domain": "numina",
        "question": "9. Stubborn Squares (from 7th grade. 2 points). Given 100 numbers. 2 was added to each of them. The sum of the squares of the numbers did not change. 2 was added to each of the resulting numbers again. How did the sum of the squares change now?",
        "answer": "800",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10785,
        "difficulty": null
      },
      {
        "id": "45e1ddec7bd6fb350859ad3721ae9109be8efc393dac8a738fe89d4cd0d86658",
        "domain": "numina",
        "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2fdcf97aa9799d0d4cd6g-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15460,
        "difficulty": null
      },
      {
        "id": "0cbfab2e097ead9190262ab4456eecda743aceface65b9fba4f98c7c13284095",
        "domain": "numina",
        "question": "## Subject III. (20 points)\n\nLet $E_{1}=\\frac{a(2 a+8)+8}{(a+4)(2 a+12)-16}$ and $E_{2}=\\frac{a(a+4)+4}{a(a+1)-2}$.\n\na) Determine $a \\in \\mathbb{Z}$ such that the expressions make sense;\n\nb) Determine $a \\in \\mathbb{Z}$ for which $\\frac{1}{E_{1}}+\\frac{1}{E_{2}} \\in \\mathbb{N}$;\n\nc) Calculate $a \\in \\mathbb{Z}$ such that $\\frac{1}{E_{1}}+\\frac{1}{E_{2}}=\\frac{3 a+10}{a+2}$.\n\nProf. Simona Pop, Colegiul Augustin Maior Cluj-Napoca",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 177800,
        "difficulty": null
      },
      {
        "id": "d021c5664f192e225ced65bfd9eeaba84c71eec7d9f8f0834ceda902b452edf3",
        "domain": "numina",
        "question": "G8.3 There are exactly $N$ values of $\\alpha$ satisfying the equation $\\cos ^{3} \\alpha-\\cos \\alpha=0$, where $0^{\\circ} \\leq \\alpha \\leq 360^{\\circ}$. Find $N$.\n\nG8.4 If the $N^{\\text {th }}$ day of May in a year is Thursday and the $K^{\\text {th }}$ day of May in the same year is Monday, where $10<K<20$, find $K$.",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152581,
        "difficulty": null
      },
      {
        "id": "127548b2dbae4865c61b133d0e9e68ccc173d889241a6dc96716d95696c6e73b",
        "domain": "numina",
        "question": "$\\left[\\begin{array}{ll}\\text { Ratios of areas } \\\\ \\text { P } & \\text { Product of lengths of chord segments and lengths of secant segments }]\\end{array}\\right]$ [ Ratio of areas of triangles with a common angle ]\n\nIn triangle $A B C$, points $B_{1}$ and $C_{1}$ are chosen on sides $A B$ and $A C$ respectively, such that $A B_{1}: A B=1: 3$ and $A C_{1}: A C=1: 2$. A circle is drawn through points $A, B_{1}$, and $C_{1}$. A line is drawn through point $B_{1}$, intersecting segment $A C_{1}$ at point $D$, and the circle at point $E$. Find the area of triangle $B_{1} C_{1} E$, if $A C_{1}=4, A D=1$, $D E=2$, and the area of triangle $A B C$ is 12.",
        "answer": "\\frac{7}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61453,
        "difficulty": null
      },
      {
        "id": "ed9bb0b972d922b6b76a61fbf2b39c9d54fe4471085e726bc3ee1d4aea48b280",
        "domain": "numina",
        "question": "【Question 8】\nThe sum of five consecutive even numbers is a multiple of 7. The smallest possible sum of these five numbers is\n\n保留了源文本的换行和格式。",
        "answer": "70",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 143687,
        "difficulty": null
      },
      {
        "id": "ef281d4499bbb5fbd6eb976912a57f3bd72685316d7b9e0d245748845f936ff0",
        "domain": "numina",
        "question": "Task 3. (15 points) Laboratory engineer Sergei received an object for research consisting of about 200 monoliths (a container designed for 200 monoliths, which was almost completely filled). Each monolith has a specific name (sandy loam or clayey loam) and genesis (marine or lake-glacial deposits). The relative frequency (statistical probability) that a randomly selected monolith will be sandy loam is $\\frac{1}{9}$. The relative frequency that a randomly selected\nmonolith will be marine clayey loam is $\\frac{11}{18}$. How many monoliths of lake-glacial genesis does the object contain, if there are no marine sandy loams among the sandy loams?",
        "answer": "77",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7261,
        "difficulty": null
      },
      {
        "id": "8149e9d1ed9b04b91546358054584939e89466e2d33efd6b0a928d69cf3cd98b",
        "domain": "numina",
        "question": "7. (10 points) In a class of 40 students all facing forward, they stand in a line from front to back, and report numbers in the order of $1, 2, 3, 4, 1, 2, 3,$\n4, 1, 2, 3, 4, and so on, with each student reporting a number once. Students who report 3 turn around. Afterwards, if two adjacent students face each other, they will shake hands once and then turn around simultaneously, until no students are facing each other. How many times do all the students shake hands in the entire process? $\\qquad$ times.",
        "answer": "145",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 114380,
        "difficulty": null
      },
      {
        "id": "cc22226e4d55af1ba91fab7024fc947edb146ce08bc02033a3abf042985c553b",
        "domain": "numina",
        "question": "1. Find the sum of all three-digit numbers that can be written using the digits $1,2,3$ such that all digits in each number are different.",
        "answer": "1332",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 42713,
        "difficulty": null
      },
      {
        "id": "3a290a046fb264367f0c84ce3cd16b001bcc899e980e9793609607269a6922c0",
        "domain": "numina",
        "question": "Task 5. (20 points) At the first deposit, equipment of the highest class was used, and at the second deposit, equipment of the first class was used, with the highest class being less than the first. Initially, $40 \\%$ of the equipment from the first deposit was transferred to the second. Then, $20 \\%$ of the equipment that ended up on the second deposit was transferred back to the first, with half of the transferred equipment being of the first class. After this, the equipment of the highest class on the first deposit was 26 units more than on the second, and the total amount of equipment on the second deposit increased by more than $5 \\%$ compared to the initial amount. Find the total amount of equipment of the first class.",
        "answer": "60",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7110,
        "difficulty": null
      },
      {
        "id": "d87e5d9ec101599effcce2ff6f7d5d20300f0eced93e3c80e9d061911627d49b",
        "domain": "numina",
        "question": "5. Among the numbers between 4000 and 7000, there are $\\qquad$ numbers that are even and have four distinct digits.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "728",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109189,
        "difficulty": null
      },
      {
        "id": "c92ff6091eee52558b68abb0f33b43679fd5495625b68a2b2f05c7b35c04c189",
        "domain": "numina",
        "question": "12. Each edge of a regular tetrahedron is trisected, and through each division point, two planes are drawn, each parallel to the two faces of the tetrahedron that do not pass through this division point. How many parts do these planes divide the tetrahedron into?\n(26th Moscow Competition Problem)",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133912,
        "difficulty": null
      },
      {
        "id": "33f54289b456745d138bac51608a1fd4523858492534bfb92335a7c6e634d355",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\frac{1+x \\cdot 2^{x}}{1+x \\cdot 3^{x}}\\right)^{\\frac{1}{x^{2}}}$",
        "answer": "\\frac{2}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46319,
        "difficulty": null
      },
      {
        "id": "bb57128174c5b12601a3c02a40b951b2c8ba910c82be34d98e3731fb85b268d1",
        "domain": "numina",
        "question": "Let $a=\\sqrt{4+\\sqrt{5-a}}, b=\\sqrt{4+\\sqrt{5+b}}, c=\\sqrt{4-\\sqrt{5-c}}$ and $d=\\sqrt{4-\\sqrt{5+d}}$.\n\nCalculate $a b c d$.",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 82894,
        "difficulty": null
      },
      {
        "id": "5015adf5e88b4b3994b2814b63f3992d1e684a41754f7865980ba3c43e110707",
        "domain": "numina",
        "question": "Task 3. (15 points) At the research institute, a scientific employee, Tatyana Vasilyevna, received an object for research containing about 150 oil samples (a container designed for 150 samples, which was almost completely filled). Each sample has certain characteristics in terms of sulfur content - either low-sulfur or high-sulfur, and density - either light or heavy. The relative frequency (statistical probability) that a randomly selected sample will be a heavy oil sample is $\\frac{2}{11}$. The relative frequency that a randomly selected sample will be a light low-sulfur oil sample is $\\frac{7}{13}$. How many high-sulfur oil samples does the object contain, if there were no low-sulfur samples among the heavy oil samples?",
        "answer": "66",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7279,
        "difficulty": null
      },
      {
        "id": "b0157de47d0e57f99e6a90ec9da458a6e452cf5601574eb5b3b5912794c282d3",
        "domain": "numina",
        "question": "4. The numbers $x$ and $y$ are such that the equalities $\\operatorname{tg} x+\\operatorname{tg} y=4$ and $3 \\sin (2 x+2 y)=\\sin 2 x \\sin 2 y$ hold. Find $\\operatorname{ctg} x \\operatorname{ctg} y$.",
        "answer": "\\frac{7}{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3504,
        "difficulty": null
      },
      {
        "id": "1cf0ff0f1a7e85d7d279ec0e7f281a9171b9597f3ba43cc219a7d68a65052bdc",
        "domain": "numina",
        "question": "2. How many ordered pairs of integers $(a, b)$ are there such that $|a|+|b|<2019$?",
        "answer": "8148685",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86474,
        "difficulty": null
      },
      {
        "id": "b9231ba0f8b66052ec5f4bff2f00ff7d52402253dfa1ffbfdc43c0a03abf0d12",
        "domain": "numina",
        "question": "11. [8] Isosceles trapezoid $A B C D$ with bases $A B$ and $C D$ has a point $P$ on $A B$ with $A P=11, B P=27$, $C D=34$, and $\\angle C P D=90^{\\circ}$. Compute the height of isosceles trapezoid $A B C D$.",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151963,
        "difficulty": null
      },
      {
        "id": "7cd53a5748c0eb319e3e1e015a61f5fa5d659069a82ffeb5b047491ae2a307e1",
        "domain": "numina",
        "question": "30. $\\mathrm{P}$ is the set $\\{1,2,3, \\ldots, 14,15\\}$. If $\\mathrm{A}=\\left\\{a_{1}, a_{2}, a_{3}\\right\\}$ is a subset of $\\mathrm{P}$ where $a_{1}<a_{2}<a_{3}$ such that $a_{1}+6 \\leq a_{2}+3 \\leq a_{3}$. How many such subsets are there of P?",
        "answer": "165",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160286,
        "difficulty": null
      },
      {
        "id": "fd1e01cdf7604459b42a6700dbfcd8a03f583881eaa93d02befe265b2f900d85",
        "domain": "numina",
        "question": "Problem 10.7. On an island, there are 23 knights and 200 liars; all residents have different names. A tourist who knows this asked each of the 223 residents to write down 200 names of liars on a piece of paper. Each knight correctly wrote down 200 names of liars, while each liar wrote down an arbitrary list of 200 names, which definitely does not include their own name. What is the maximum number of liars the tourist can definitely identify from these data?",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 18014,
        "difficulty": null
      },
      {
        "id": "aa5bfffbb0fcbc2bb31f50fc75a1cf302db8ae33855eae508841afa8cc4f5dd3",
        "domain": "numina",
        "question": "91. On a long strip of paper, consecutive natural numbers $1, 2, 3, \\cdots \\cdots n$ are written in sequence. The long strip is cut into five segments, each containing some numbers (the digits of a single number will not be cut into different segments). The average of the numbers in these five segments are $1234, 345, 128, 19$, and $9.5$ (the order of these five numbers is scrambled). Therefore, $n=$",
        "answer": "2014",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94168,
        "difficulty": null
      },
      {
        "id": "c22ee8f8e29eed1f5d4976b619f2b6ae1b61c4afc9051478db36fd3afae0d047",
        "domain": "numina",
        "question": "How many multiples of 3 between 1 and 2015 have a units digit of 2 in their decimal representation?",
        "answer": "67",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78470,
        "difficulty": null
      },
      {
        "id": "64b03a30e72cc642f8e19949c7e3a6ea8e032995dfeba9472a974ffd6f04ff49",
        "domain": "numina",
        "question": "4. From the 2022 natural numbers 1 to 2022, select several distinct numbers such that the sum of any 2 of them is not a multiple of 5. The maximum number of numbers that can be selected is $\\qquad$.",
        "answer": "811",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 123464,
        "difficulty": null
      },
      {
        "id": "81ade4373b6262a1bfc75a7a47c697e5c5a6736a8670c163340152b5b997d72e",
        "domain": "numina",
        "question": "Let's determine the pairs of squares of two-digit (integer) numbers that differ only in the hundreds place (for example, 2401 and 2601).",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 172463,
        "difficulty": null
      },
      {
        "id": "c18a4fe9d1450d85af002e4c566823a523d4974e31e7fd4eb39c8425aac390cb",
        "domain": "numina",
        "question": "5. (10 points) On the blackboard, there is a sequence of numbers: $1,2,3, \\cdots, 2011$. If every time the first 16 numbers are erased, and the sum of the erased 16 numbers is written at the end of the sequence, until only one number remains, then:\n(1) What is the last remaining number?\n(2) What is the total sum of all numbers that have appeared on the blackboard?",
        "answer": "7822326",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 145543,
        "difficulty": null
      },
      {
        "id": "30522f5c2fb506c61b592c41a70a98dc750ed826b8ce5ae3294243d033ccb65d",
        "domain": "numina",
        "question": "7. Given that the complex number $z$ satisfies $|z|=1$, then the maximum value of $\\left|z^{3}-3 z-2\\right|$ is $\\qquad$ .",
        "answer": "3\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141011,
        "difficulty": null
      },
      {
        "id": "b98af9530b81f499b8e7282803cd1878f60f683cdf72f1c07bedd5b3c93d35ec",
        "domain": "numina",
        "question": "The third-grade students who participated in the 11th \"Math Walk\" and entered the finals were arranged in a square formation, with 15 remaining, and at least 18 more people are needed to form a larger square formation. There are $\\qquad$ third-grade students participating in the 11th \"Math Walk\" competition.",
        "answer": "271",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 133521,
        "difficulty": null
      },
      {
        "id": "c15f8027c7729e2547a2990e5513194a5b6d433347467708e6666c5eb718057e",
        "domain": "numina",
        "question": "7. The commission consists of 11 people. The materials that the commission works on are stored in a safe. How many locks should the safe have, and how many keys should each member of the commission be provided with, so that access to the safe is possible when a majority of the commission members gather, but not possible if fewer than half of the members gather?",
        "answer": "252",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 23886,
        "difficulty": null
      },
      {
        "id": "ae0e26716640c97a4d2bd32b54ae43ee5b021b47caaf6c88889eaf278d994800",
        "domain": "numina",
        "question": "4.8 There is a four-digit number less than 2000, which has exactly 14 positive divisors (including 1 and itself), and one of its prime divisors ends with the digit 1. Find this four-digit number.\n(Shanghai Junior High School Mathematics Competition, 1984)",
        "answer": "1984",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136110,
        "difficulty": null
      },
      {
        "id": "637ea9fc0419d58e05e8c50926ec1b6b84cf682459a9db7b56d13a89cf331b99",
        "domain": "numina",
        "question": "8. 确定 $\\sqrt{25^{\\frac{1}{\\log _{6} 5}}+49^{\\frac{1}{\\log _{8} 7}}}$ 的值.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127002,
        "difficulty": null
      },
      {
        "id": "9d4b9ad0b20fb475fb8281b73b56cf7c804782f6d82b3b3175c3605df15a16ed",
        "domain": "numina",
        "question": "1. The product of all positive divisors of a natural number $n$ is $20^{15}$. Determine $n$.",
        "answer": "400",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75779,
        "difficulty": null
      },
      {
        "id": "9e9a2512a511c75ca3297f2f4ff70228296c3dc456274d0947b66540d0ff017f",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the figure bounded by the graphs of the functions:\n\n$$\nx=4-(y-1)^{2}, x=y^{2}-4 y+3\n$$",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47524,
        "difficulty": null
      },
      {
        "id": "16c8cc7b61b0454a2bebb67a97ebf8380d2bfa279fba120045d9bcf7e8a2a125",
        "domain": "numina",
        "question": "31. Let's call a number complex if it has at least two different prime divisors. Find the largest natural number that cannot be represented as the sum of two complex numbers.",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 18883,
        "difficulty": null
      },
      {
        "id": "cbed1f8c798fa45b652a197912d313af9f1ec4d1c96480ade1cf6797f1228897",
        "domain": "numina",
        "question": "4. In the rectangular prism $A^{\\prime} C$, $A B=5, B C=4, B^{\\prime} B=6$, and $E$ is the midpoint of $A A^{\\prime}$. Find the distance between the skew lines $B E$ and $A^{\\prime} C^{\\prime}$.",
        "answer": "\\frac{60}{\\sqrt{769}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93049,
        "difficulty": null
      },
      {
        "id": "fc558c8aaf0479a3a837741df44c16b21fa82831cf22656f3766b5ea3e9a95d4",
        "domain": "numina",
        "question": "8. Jon created a unique encryption method, where each digit of the plaintext number is transformed into a digit of the ciphertext number by \"multiplying by 3, adding 1, and taking the unit digit,\" for example, the plaintext \"27\" is encrypted to the ciphertext \"72\". If a four-digit plaintext number, after two rounds of encryption, becomes the ciphertext \"2019\", then this four-digit plaintext number is $\\qquad$",
        "answer": "2435",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124666,
        "difficulty": null
      },
      {
        "id": "f3dbeba51c15f370875eb155cad4b8c2360bf5e66a7619ef6a4f0d5bb4d0cfb4",
        "domain": "numina",
        "question": "Solve the following equation:\n\n$$\n2^{x+3}+4^{x+1}=320 .\n$$",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 176012,
        "difficulty": null
      },
      {
        "id": "ad6c87f93a8a3229237a90b8f1fa5b618f13c29b6c0cdfe37e93eff7a0f99cc5",
        "domain": "numina",
        "question": "Raskin M.A.\n\nMasha has two-ruble and five-ruble coins. If she takes all her two-ruble coins, she will be 60 rubles short to buy four pies. If all five-ruble coins - she will be 60 rubles short for five pies. And in total, she is 60 rubles short to buy six pies. How much does a pie cost?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 72375,
        "difficulty": null
      },
      {
        "id": "77179965a355e64404ea6e4eeedf91154dd565b355f3b6b716d0f70230f7b7f2",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{-\\pi / 2}^{0} 2^{8} \\cdot \\cos ^{8} x d x\n$$",
        "answer": "35\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46302,
        "difficulty": null
      },
      {
        "id": "5eacf8aa87fc8426a6d9d9fe6ff1213c93f9235938545e012d3445e8a3d2d0fd",
        "domain": "numina",
        "question": "2. The cold water tap fills the bathtub in 19 minutes, while the hot water tap fills it in 23 minutes. The hot water tap was opened. After how many minutes should the cold water tap be opened so that by the time the bathtub is completely filled, there is an equal amount of cold and hot water in it?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4267,
        "difficulty": null
      },
      {
        "id": "27ff63e5bdcc41ab9f534b1f97fc5842c09a20bdcdbc8ce815d0d649c7ad63cf",
        "domain": "numina",
        "question": "8. Given that the side faces of a regular tetrahedron are right triangles with an area of 1, then the volume of the regular tetrahedron is",
        "answer": "\\frac{\\sqrt{2}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 94758,
        "difficulty": null
      },
      {
        "id": "c20635ac2f764e3183df6e0014a8312a07865c5072dc7cc07fdf49b314ca6fec",
        "domain": "numina",
        "question": "2. We consider number sequences $a_{1}, a_{2}, a_{3}, \\ldots$ such that $a_{n+1}=\\frac{a_{n}+a_{1}}{a_{n}+1}$ holds for all $n \\geqslant 1$.\n\n(a) Suppose that $a_{1}=-3$. Compute $a_{2020}$.\n\n(b) Suppose that $a_{1}=2$. Prove that $\\frac{4}{3} \\leqslant a_{n} \\leqslant \\frac{3}{2}$ holds for all $n \\geqslant 2$.",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 85736,
        "difficulty": null
      },
      {
        "id": "047e156c3c5ef2d6ff1d86fe29a31804b5cbdb4a6f08dc1e938cb3eab87c97e4",
        "domain": "numina",
        "question": "10. Suppose $\\omega$ is a circle centered at $O$ with radius 8 . Let $A C$ and $B D$ be perpendicular chords of $\\omega$. Let $P$ be a point inside quadrilateral $A B C D$ such that the circumcircles of triangles $A B P$ and $C D P$ are tangent, and the circumcircles of triangles $A D P$ and $B C P$ are tangent. If $A C=2 \\sqrt{61}$ and $B D=6 \\sqrt{7}$, then $O P$ can be expressed as $\\sqrt{a}-\\sqrt{b}$ for positive integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "103360",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151937,
        "difficulty": null
      },
      {
        "id": "0261ae5c2dbc930d163007773cd700e0359d15903f86a2cc0dfb0730ccc0c8d1",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the parametric equations.\n\n$$\n\\begin{aligned}\n& \\left\\{\\begin{array}{l}\nx=\\left(t^{2}-2\\right) \\sin t+2 t \\cos t \\\\\ny=\\left(2-t^{2}\\right) \\cos t+2 t \\sin t\n\\end{array}\\right. \\\\\n& 0 \\leq t \\leq 3 \\pi\n\\end{aligned}\n$$",
        "answer": "9\\pi^{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47552,
        "difficulty": null
      },
      {
        "id": "213e48b8e6663d5c89a2852395d630b578ac14cc1206fe640cb0955129ff5661",
        "domain": "numina",
        "question": "7. Find the sum of all possible different two-digit numbers, all digits of which are odd.",
        "answer": "1375",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 42676,
        "difficulty": null
      },
      {
        "id": "4e2384da8ff6b024b0fb306a2c7d90304682b4151551dd9f8901365770f4479c",
        "domain": "numina",
        "question": "2. Batman wants to crack Edward Nigma's code. He knows that the code represents some permutation of the letters in the expression TRICKORTREAT, and that the first and last letters of the code are the same. How many possibilities are there for such a code?",
        "answer": "\\frac{10!}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 161314,
        "difficulty": null
      },
      {
        "id": "ce57002bc0dc91f7269651cb4a5a203ba149e319d89004bd372c930189e91850",
        "domain": "numina",
        "question": "Megrajzoltuk a koordinátarendszerben az $f(x)=\\frac{1}{x}$ függvény grafikonját. Mekkorának válasszuk az új, egymással továbbra is egyenlő egységeket a tengelyeken, ha azt akarjuk, hogy a görbe a $g(x)=\\frac{2}{x}$ függvény grafikonja legyen?",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171472,
        "difficulty": null
      },
      {
        "id": "08a509b18944ed735bae6e276c56434ff7265cbf7702a04b616b2aa1f2325462",
        "domain": "numina",
        "question": "4. (10 points) A school has two classes each in the third and fourth grades. Class 3-1 has 4 more students than Class 3-2, Class 4-1 has 5 fewer students than Class 4-2, and the third grade has 17 fewer students than the fourth grade. Therefore, Class 3-1 has fewer students than Class 4-2 by ____.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111377,
        "difficulty": null
      },
      {
        "id": "64372ea72f1ccf02f2e2a32949ded560367bebf741d885c91a805ace0125f86b",
        "domain": "numina",
        "question": "8. If a positive integer $n$ makes the equation $x^{3}+y^{3}=z^{n}$ have positive integer solutions $(x, y, z)$, then $n$ is called a \"good number\". Then the number of \"good numbers\" not exceeding 2019 is $\\qquad$ .",
        "answer": "1346",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 116843,
        "difficulty": null
      },
      {
        "id": "921f0c39c2e0467d58c1aee8542c97d7257b34153a3ddc51659dc12715423dfe",
        "domain": "numina",
        "question": "2. (5 points) Xiao Fou buys 3 erasers and 5 pencils for 10.6 yuan. If he buys 4 erasers and 4 pencils of the same type, he needs to pay 12 yuan. Then the price of one eraser is $\\qquad$ yuan.",
        "answer": "2.2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138730,
        "difficulty": null
      },
      {
        "id": "1dc9dcf1382bc3816575b71f0d5c204d8e52bb328ed25f53c9459bd0e0ac7d45",
        "domain": "numina",
        "question": "The distance from a vertex of the base of a regular tetrahedron to the centroid of the opposite side is 4. The maximum volume of this regular tetrahedron is\n$\\qquad$",
        "answer": "11.18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92309,
        "difficulty": null
      },
      {
        "id": "a6ec73774765fe6649201c3c146856dfd63e4c7b685f0438b40aeceeea490656",
        "domain": "numina",
        "question": "4. The base of a quadrilateral pyramid is a square $ABCD$, with a side length of 2, and the lateral edge $SA$ is perpendicular to the base plane and also equals 2. A plane is drawn through the lateral edge $SC$ and a point on the side $AB$, such that the resulting cross-section of the pyramid has the smallest perimeter. Find the area of the cross-section.",
        "answer": "\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14996,
        "difficulty": null
      },
      {
        "id": "086937c96fb074745b6ce176f3b9a0b270b5a01b9fc808f2883635529392b21c",
        "domain": "numina",
        "question": "(15) Let $x, y$ be real numbers, then $\\max _{5 x^{2}+4 y^{2}=10 x}\\left(x^{2}+y^{2}\\right)=$ $\\qquad$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110137,
        "difficulty": null
      },
      {
        "id": "ee06ed6a6dc9f2571e9ca6da14f9186935a7f0c79ae8c3a42739bfd4b093d4c2",
        "domain": "numina",
        "question": "## Task 1 - 210711\n\nThe FDJ members of a school have decided to redesign the grounds of their school. A rectangular lawn area measuring $45 \\mathrm{~m}$ in length and $26 \\mathrm{~m}$ in width is to be surrounded by a path that is $2 \\mathrm{~m}$ wide.\n\nThe path is to run outside the lawn and adjoin it all around. The area occupied by the lawn and the path together is to form a rectangle again.\n\na) Calculate the area of the planned path!\n\nb) How many paving stones are needed for the path in total if it is to be completely covered with paving stones and if exactly 16 stones are required for each square meter of the path?",
        "answer": "4800",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 183715,
        "difficulty": null
      },
      {
        "id": "a416752d7f68c35b7fb6969bdd0f238af25e811e8d05f1974f714c5d8464f13f",
        "domain": "numina",
        "question": "Problem 4.5. Hooligan Dima laid out a structure in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the diagram.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire structure to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_df4c0436ab68970fab15g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
        "answer": "65",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16304,
        "difficulty": null
      },
      {
        "id": "16e275b10ea9c8d3e4578aab6f2f0c05fcd96d0a3a9dbd8e58b8a6e3c1800d75",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\frac{\\arcsin ^{2} x}{\\arcsin ^{2} 4 x}\\right)^{2 x+1}$",
        "answer": "\\frac{1}{16}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46030,
        "difficulty": null
      },
      {
        "id": "263b59478f98c3a55f3dd970fd6e64a094588b340de8f629de1a7cc3d14327cf",
        "domain": "numina",
        "question": "Example 1. Calculate approximately the value of the function $f(x)=$ $=\\sqrt{x^{2}+x+3}$ at $x=1.97$",
        "answer": "2.975",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 30243,
        "difficulty": null
      },
      {
        "id": "ef23066135c13b8727084f45de2b912db09f92e7781cc6da8dfa29e000249dbf",
        "domain": "numina",
        "question": "In the following figure-not drawn to scale!- $E$ is the midpoint of $B C$, triangle $F E C$ has area 7, and quadrilateral $D B E G$ has area 27. Triangles $A D G$ and $G E F$ have the same area, $x$. Find $x$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_6cb05c45fe7c141f3316g-4.jpg?height=661&width=700&top_left_y=1209&top_left_x=688)",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 90914,
        "difficulty": null
      },
      {
        "id": "bf5c1a85096f246a9bd596819ef57fa18c384a90d2a0aefccf7bc482700cbac9",
        "domain": "numina",
        "question": "On each kilometer of the highway between the villages of Yolki and Palki, there is a post with a sign. On one side of the sign, it says how many kilometers it is to Yolki, and on the other side - to Palki. A Thoughtful Observer noticed that on each post, the sum of these numbers is 13. What is the distance from Yolki to Palki?\n\n#",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69058,
        "difficulty": null
      },
      {
        "id": "1f638653350b952b7a720cf320e112ef47331abf815e6c6957d09ed6257ba042",
        "domain": "numina",
        "question": "1. When $3 x^{2}$ was added to the quadratic trinomial $f(x)$, its minimum value increased by 9, and when $x^{2}$ was subtracted from it, its minimum value decreased by 9. How will the minimum value of $f(x)$ change if $x^{2}$ is added to it?",
        "answer": "\\frac{9}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3112,
        "difficulty": null
      },
      {
        "id": "fef204a31cb8331188ec352f7525ef6bd6757f02f5fc7a0086e2b4938bdd1385",
        "domain": "numina",
        "question": "In a magic square, the sum of the numbers in each row, column, and diagonal is the same. In the magic square below, what is the value of $a+b+c$?\n\n| 16 | 2 | $a$ |\n| :---: | :---: | :---: |\n| $c$ | 10 | $d$ |\n| $b$ | $e$ | 4 |",
        "answer": "26",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 73339,
        "difficulty": null
      },
      {
        "id": "4765aaaffd8bbe3b284290b137432a533c9828009527f6f97191862eb2c17bb0",
        "domain": "numina",
        "question": "7. (5 points) This year is 2017, the sum of the digits of the year is 10, then within this century, the sum of all years whose digit sum is 10\n\n保留源文本的换行和格式，翻译结果如下：\n\n7. (5 points) This year is 2017, the sum of the digits of the year is 10, then within this century, the sum of all years whose digit sum is 10",
        "answer": "18396",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105882,
        "difficulty": null
      },
      {
        "id": "01c1aecde9c7d58dc58d9030ad25d19e77f3ddda6f9381036c9db28c0921f6b7",
        "domain": "numina",
        "question": "In a quadrilateral $A B C D$, a circle of radius 2 is inscribed. The angle $\\angle D A B$ is a right angle. Side $A B$ is 5, and side $B C$ is 6. Find the area of the quadrilateral $A B C D$.",
        "answer": "17\\frac{11}{17}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63039,
        "difficulty": null
      },
      {
        "id": "8e676e5cf533e308776d2905741c51729559137c6e794a5915b73833d83e7578",
        "domain": "numina",
        "question": "1.59 (1) $f(x)$ is a monovariate quartic polynomial with integer coefficients, i.e.,\n$$\nf(x)=c_{4} x^{4}+c_{3} x^{3}+c_{2} x^{2}+c_{1} x+c_{0},\n$$\n\nwhere $c_{0}, c_{1}, c_{2}, c_{3}, c_{4}$ are all integers. Prove that if $a, b$ are integers and $a > b$, then $a - b$ divides $f(a) - f(b)$.\n(2) Two people, A and B, are doing homework in the classroom.\nA asks B: “How many problems have you finished?”\nB replies: “The number of problems I have finished is a positive integer and is a root of a monovariate quartic polynomial with integer coefficients. Can you guess it?”\nThen, A tries substituting 7 into the polynomial and gets a value of 77. At this point, B says: “The number of problems I have finished is more than 7.”\nA says: “Alright, I will try a larger integer.” So A substitutes $B$ into the polynomial and gets a value of 85.\nB looks and says: “The number of problems I have finished is more than $B$.” After some thought and calculation, A finally determines the number of problems B has finished.\nTry to find the number of problems B has finished based on the conversation.\n(China Hebei Province Mathematics Competition, 1979)",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 121588,
        "difficulty": null
      },
      {
        "id": "48ef0cbdb16af415da66b5ac4fd380f4ec24f31ddc894e601f1246e71ad7e743",
        "domain": "numina",
        "question": "Seven teams, divided into two groups, participated in the football tournament in my neighborhood. Group 1 consisted of the teams Avaqui, Botágua, and Corinense. Group 2 consisted of the teams Dinossaurs, Esquisitos, Flurinthians, and Guaraná.\n\n$\\mathrm{In}$ the first round of the tournament, each team played against each of the other teams in its group exactly once. In the second round of the tournament, each team played against each of the teams in the other group exactly once.\n\n(a) How many matches were played in the first round in Group 1?\n\n(b) How many matches were played in the first round in Group 2?\n\n(c) How many matches were played in the second round?",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74305,
        "difficulty": null
      },
      {
        "id": "3647b8001a2802dff2842783d8d60f51cf34e9af502a857d1a19cea556f806f4",
        "domain": "numina",
        "question": "9. Given $F_{1}$ and $F_{2}$ are the two foci of the ellipse $\\frac{x^{2}}{4}+y^{2}=1$, $A$ and $B$ are the left vertex and the upper vertex of the ellipse, respectively, and point $P$ lies on the line segment $A B$. Then the minimum value of $\\overrightarrow{P F_{1}} \\cdot \\overrightarrow{P F_{2}}$ is .",
        "answer": "-\\frac{11}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92765,
        "difficulty": null
      },
      {
        "id": "7fa7dd3ac106a0f6e974021af9918afd2fc155680dcda783d124dcac124606c2",
        "domain": "numina",
        "question": "2. Painting walls in a new building 10 workers would complete in 15 days. However, at the beginning, there were only 6 workers, and after 5 days, 2 more workers joined, and after another 3 days, 4 more workers joined. In how many days was the job finally completed?",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86966,
        "difficulty": null
      },
      {
        "id": "8fa74e9fcded81f26ba3f3b3d1af9f9c1a7dc4fade0621c3da69f8862b91ce05",
        "domain": "numina",
        "question": "I4.4 Let $D$ be the sum of the last three digits of $2401 \\times 7^{C}$ (in the denary system). Find the value of $D$.",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153295,
        "difficulty": null
      },
      {
        "id": "2f651d3cd26b2e1d3604eb3f9e69f229181bd04da819d72f8b25d3c6451928aa",
        "domain": "numina",
        "question": "11. [15] Find the largest positive integer $n$ such that $1!+2!+3!+\\cdots+n$ ! is a perfect square. Prove that your answer is correct.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152441,
        "difficulty": null
      },
      {
        "id": "60a60b3ee7b9848d476a47f8ff09cb96d3a1109def2594d1227df1aa144eb98f",
        "domain": "numina",
        "question": "2. The sequence $\\left(a_{n}\\right)$ is defined by the following relations: $a_{1}=1, a_{2}=2, a_{n}=a_{n-1}-a_{n-2}+n$ (for $n \\geqslant 3$). Find $a_{2019}$.",
        "answer": "2020",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3976,
        "difficulty": null
      },
      {
        "id": "85a078b624404f42ae41d1a70e40f3060765310bd18dfd5eed903b077322c3aa",
        "domain": "numina",
        "question": "Pedro has a $6 \\times 2$ board, with the two rightmost cells painted gray as illustrated below:\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d1e5bbfbdffc7558a598g-11.jpg?height=222&width=616&top_left_y=1882&top_left_x=797)\n\nHe must fill all the cells of his board with the numbers from 1 to 12 such that:\n\n- in each row, the 6 numbers, read from left to right, are in increasing order; and\n- in each column, the number on top is less than the number below.\na) Show that the largest sum Pedro can achieve in the gray cells is 23. Also show how he can achieve this sum.\n\nb) Show that the smallest sum Pedro can achieve in the gray cells is 18. Also show how he can achieve this sum.\n\n#",
        "answer": "18",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73432,
        "difficulty": null
      },
      {
        "id": "b8a3dda854a8376f109637be7744357e9897f331ae2fbe875e9715251131ccc2",
        "domain": "numina",
        "question": "Subject 2. The arithmetic mean of the distinct natural numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$, 2014, and 2016 is 2015.\n\na. Find the arithmetic mean of the numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$.\n\nb. If the numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$ are even, show that $a_{1} \\cdot a_{2} \\cdot a_{3} \\cdot \\ldots \\cdot a_{2014}=0$",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 178353,
        "difficulty": null
      },
      {
        "id": "1906bb584cac56e51e8b0f794b62dfbd13fdf5eefba5e087071ac4dae6a42668",
        "domain": "numina",
        "question": "14. (6th \"Hope Cup\" Senior High School Competition) The function $y=f(x)$ is an even function and is a periodic function with a period of 2. When $x \\in [2,3]$, $f(x)=x-1$. On the graph of $y=f(x)$, there are two points $A$ and $B$, whose y-coordinates are equal ($A$ is to the left of $B$), and their x-coordinates are in the interval $[1,3]$. The fixed point $C$ has coordinates $(0, a)$, where $a>2$. Find the maximum value of the area of $\\triangle ABC$ (expressed in terms of $a$).",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100018,
        "difficulty": null
      },
      {
        "id": "b7ad2d5507dbce71184832d7d6e1789ffde2d6058e8fec78138fb3f04c139587",
        "domain": "numina",
        "question": "1B. If the digit 1 is appended to the right of a three-digit number, a four-digit number is obtained which is 3 times larger than the number obtained by appending the digit 2 to the left of the given three-digit number. What is that three-digit number?",
        "answer": "857",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190496,
        "difficulty": null
      },
      {
        "id": "3196605edb326030de4215b24796048b2d0a2bd45b816cba4f71a7d465a7c4d3",
        "domain": "numina",
        "question": "Auxiliary similar triangles Pythagorean theorem (direct and inverse).\n\nIn triangle $ABC$, altitudes $AA_1$ and $BB_1$ are drawn. Find $AC$, if\na) $AA_1=4, BB_1=5, BC=6$\n\nb) $A_1C=8, B_1C=5, BB_1=12$.",
        "answer": "4.8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63298,
        "difficulty": null
      },
      {
        "id": "90fabed9242c5a428b19c14ebb194be05321c642886453636e24d64bf9f2de3a",
        "domain": "numina",
        "question": "1. The engines of a rocket launched vertically upward from the Earth's surface, providing the rocket with an acceleration of $20 \\mathrm{~m} / \\mathrm{c}^{2}$, suddenly stopped working 40 seconds after launch. To what maximum height will the rocket rise? Can this rocket pose a danger to an object located at an altitude of 45 km? Air resistance is not taken into account, and the acceleration due to gravity is considered to be $10 \\mathrm{~m} / \\mathrm{c}^{2}$.",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11217,
        "difficulty": null
      },
      {
        "id": "6e63d19222e5fa20937fe8184c14da8553de9957527c67760867c79122a80117",
        "domain": "numina",
        "question": "## Task 2 - 330622\n\nImagine all the different numbers formed by arranging the five digits 1, 2, 3, 4, 5 in every possible order.\n\nWhat is the sum of all these five-digit numbers?",
        "answer": "3999960",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 183420,
        "difficulty": null
      },
      {
        "id": "cf0b99f8545c2e7f438d6834f030f4721316b56dc900196a4ab05cdb5332052a",
        "domain": "numina",
        "question": "4. (10 points) In a lottery, 6 numbers out of 36 are drawn each week. What is the probability that at least one of the numbers drawn this week will be the same as one of the numbers drawn last week? Round the result to the nearest thousandth.",
        "answer": "0.695",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19771,
        "difficulty": null
      },
      {
        "id": "2da1f3db0438fdba4f4bcc23363e24aa4aef15fd4bebfd747fb529b879849e56",
        "domain": "numina",
        "question": "2. (5 points) Calculate: $898+9898+99898+999898=$",
        "answer": "1110592",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 126337,
        "difficulty": null
      },
      {
        "id": "6e1f5ead875ccd035f8facc32e0678abf1d24b128d91df285135d0b148d1c9a5",
        "domain": "numina",
        "question": "Task B-1.1. Solve the equation:\n\n$$\n|x-| x-|x+1|||=x\n$$",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87678,
        "difficulty": null
      },
      {
        "id": "44b87e3a1b001472bd226a1143bcc296ee79db21ee9b7d82b292d3df96d4746f",
        "domain": "numina",
        "question": "132. For which $x$ and $y$ is the number $x x y y$ a square of a natural number?",
        "answer": "7744",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 36056,
        "difficulty": null
      },
      {
        "id": "e8fa712e47b06eba7abb900fbaf4300627adf0d51828c2f0f657dde10429cceb",
        "domain": "numina",
        "question": "Example 12 Find the minimum value of $\\frac{3}{\\cos x}+\\frac{2}{\\sin x}\\left(0<x<\\frac{\\pi}{2}\\right)$.",
        "answer": "\\sqrt{(\\sqrt[3]{9}+\\sqrt[3]{4})^{3}}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 111354,
        "difficulty": null
      },
      {
        "id": "3ea45854ee7cbc38835df48ae7aa3a1e717c65ce05c0d71d33b9dc508f8472c8",
        "domain": "numina",
        "question": "Problem 4. Given is a square $ABCD$ with side length $9 \\, \\text{cm}$. The square is folded as shown in the diagram on the right, so that vertex $B$ coincides with point $E$, which lies on side $\\overline{CD}$. In this way, three triangles are formed: $CEF$, $EDG$, and $GAH$. What is the sum of the perimeters of these three triangles?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_e235c78af50361de0ec1g-04.jpg?height=272&width=348&top_left_y=1859&top_left_x=1114)",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188727,
        "difficulty": null
      },
      {
        "id": "03df357fa782474eff9278f33240b69759c9503d76b23f3411ea0411d5dd1da8",
        "domain": "numina",
        "question": "3. (17th Japan Mathematical Olympiad) Find the tens digit of $11^{12^{13}}$ (where $11^{12^{3}}$ represents 11 to the power of $12^{13}$)",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 135740,
        "difficulty": null
      },
      {
        "id": "d2dd661c1d72143b47ab702a720d047561e3269b982684fabd7ed04de2a6dd1d",
        "domain": "numina",
        "question": "It is said that an integer $n \\geqslant 1$ is Japanese if it admits three divisors $d_{1}, d_{2}$, and $d_{3}$ such that $1 \\leqslant d_{1}<d_{2}<d_{3}$ and $d_{1}+d_{2}+d_{3}=2022$.\nWhat is the smallest Japanese integer?",
        "answer": "1344",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 195794,
        "difficulty": null
      },
      {
        "id": "82fc66967ca49645da867a1b8c95f57b819579faa10f9f6d7de6518d479a0ee9",
        "domain": "numina",
        "question": "Task A-1.4. (4 points)\n\nIf we delete the units digit of a number, we will get a number that is 2010 less than the original number. What is the original number?",
        "answer": "2233",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 88318,
        "difficulty": null
      },
      {
        "id": "dce58ac980cd594f9dc46093867de84fb0e9f34f0a62f818eb909e3c008ff718",
        "domain": "numina",
        "question": "5. (8 points) In $\\triangle A B C, A B=86$, and $A C=97$. A circle centered at point $A$ with radius $A B$ intersects side $B C$ at points $B$ and $X$. Additionally, $B X$ and $C X$ have integer lengths. What is the length of $B C ?$",
        "answer": "61",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19653,
        "difficulty": null
      },
      {
        "id": "f9f94657777007cae60a8ff4733fc68b95a294e1c92da92a6ab5f4a1e207e696",
        "domain": "numina",
        "question": "In a picture trading, the price of the frames is directly proportional to the value of the paintings inside them. To reduce the price difference between certain paintings, the dealer exchanges the frames of two paintings. In one case, a painting that cost five times as much as the other, after the frames were exchanged, only cost three times as much. How will the price ratio of the paintings titled \"Winter Landscape\" and \"Village Road\" change if, before the frames were exchanged, the \"Winter Landscape\" cost nine times as much as the \"Village Road\"?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171462,
        "difficulty": null
      },
      {
        "id": "3baa61113e375997080823abcd1ba000d0dbdd569aa757abd0afaae88dc45083",
        "domain": "numina",
        "question": "5. Sabine has a very large collection of shells. She decides to give part of her collection to her sister.\n\nOn the first day, she lines up all her shells. She takes the shells that are in a position that is a perfect square (the first, fourth, ninth, sixteenth, etc. shell), and gives them to her sister. On the second day, she lines up her remaining shells. Again, she takes the shells that are in a position that is a perfect square, and gives them to her sister. She repeats this process every day.\n\nThe 27th day is the first day that she ends up with fewer than 1000 shells. The 28th day she ends up with a number of shells that is a perfect square for the tenth time.\n\nWhat are the possible numbers of shells that Sabine could have had in the very beginning?",
        "answer": "2020",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85740,
        "difficulty": null
      },
      {
        "id": "800e65bdd07532df5344696e2456fc403165c8287c188e914771cc09970bc926",
        "domain": "numina",
        "question": "5. If the six edges of a tetrahedron are all powers of 2, then in this tetrahedron, the lengths of the edges can have at most $\\qquad$ values.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101993,
        "difficulty": null
      },
      {
        "id": "60693f720bbe6c744b7333cc3c59b173e8cc27e59c43653e5ad3df7232067a14",
        "domain": "numina",
        "question": "3. Calculate the value of the expression $\\sin ^{8} 75^{\\circ}-\\cos ^{8} 75^{\\circ}$.",
        "answer": "\\frac{7\\sqrt{3}}{16}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162702,
        "difficulty": null
      },
      {
        "id": "0273c2b39bfea847bfd230ef9d36a767e1528f1df5e11175d5fca963c6335417",
        "domain": "numina",
        "question": "In a large box, 10 smaller boxes were placed. In each of the nested boxes, either 10 even smaller ones were placed, or nothing was placed. In each of the smaller ones, either 10 or none were placed again, and so on. After this, it turned out that there were exactly 2006 boxes with contents. How many are empty? #",
        "answer": "18055",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56564,
        "difficulty": null
      },
      {
        "id": "ee24b31de4a0565749e886827fa9b4acbb7feb0133eae8541599da6eec1787d6",
        "domain": "numina",
        "question": "Anya, Borya, and Vasya decided to go to the \"Fir Tree\". They agreed to meet at the bus stop, but they don't know who will arrive at what time. Each of them can arrive at a random time between 15:00 and 16:00. Vasya is the most patient of all: if he arrives and there is neither Anya nor Borya at the bus stop, he will wait for 15 minutes, and if he doesn't wait for anyone, he will go to the \"Fir Tree\" alone. Borya is less patient: he will wait only 10 minutes. Anya is the most impatient: she won't wait at all. However, if Borya and Vasya meet, they will wait for Anya until 16:00. What is the probability that they will all go to the \"Fir Tree\" together?",
        "answer": "0.124",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55923,
        "difficulty": null
      },
      {
        "id": "af47c650b01b9bdc88b218c3b85ba30898dea19caaf45df433863b2d7a3785ba",
        "domain": "numina",
        "question": "3. Find the smallest natural number $N>9$, which is not divisible by 7, but if any of its digits is replaced by 7, the resulting number is divisible by 7.",
        "answer": "13264513",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 23012,
        "difficulty": null
      },
      {
        "id": "4de73190204cf275657ab33133c35363446a448eed4fd7e2fc5ae49930d31533",
        "domain": "numina",
        "question": "Example 12 Let $x, y, z$ be real numbers, not all zero, find the maximum value of $\\frac{x y+2 y z}{x^{2}+y^{2}+z^{2}}$.",
        "answer": "\\frac{\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129512,
        "difficulty": null
      },
      {
        "id": "7141efc88a1a3bb4feeda5d2c8d30a12aff94b4be72efa900788c74fcf9ea0cd",
        "domain": "numina",
        "question": "Example 11. Find the mathematical expectation of the random variable\n$X$, the density of which is determined by the function $p(x)=0.2 e^{-0.2 x}$ for $x \\geq 0$.",
        "answer": "5",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33113,
        "difficulty": null
      },
      {
        "id": "57434b1f529b941815339eb5c6d2fbc9d2297d5d64b6962c59a07c836be315e5",
        "domain": "numina",
        "question": "Example 4 Let the set $A=\\{1,2, \\cdots, 10\\},$ and the mapping $f$ from $A$ to $A$ satisfies the following two conditions:\n(1) For any $x \\in A, f_{30}(x)=x$.\n(2) For each $k \\in \\mathbf{N}_{+}, 1 \\leqslant k \\leqslant 29$, there exists at least one $a \\in A$ such that $f_{k}(a) \\neq a$.\n\nFind the number of such mappings $f$ (it is agreed that $f_{1}(x)=f(x), f_{k+1}(x)=f\\left(f_{k}(x)\\right), k=1,2, \\cdots$).",
        "answer": "120960",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123624,
        "difficulty": null
      },
      {
        "id": "5db2c8bee2d7af9b782f311da8fb94914426a1904e999cde809935543a1404ac",
        "domain": "numina",
        "question": "35. Suppose the three sides of a triangular field are all integers, and its area equals the perimeter (in numbers). What is the largest possible area of the field?",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160575,
        "difficulty": null
      },
      {
        "id": "7badf2caa675507558aa0ef04a178faefa1144be3d945b73cb22b72cf5632b49",
        "domain": "numina",
        "question": "11. In the multiplication equation in the figure, different Chinese characters represent different digits, and the same Chinese characters represent the same digit. Fill in the appropriate digits in the squares of the equation to make the equation true. Then, the three-digit number represented by \"中环杯\" is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "124",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 103507,
        "difficulty": null
      },
      {
        "id": "bbb9bccfb8f1ef46fa8330529583e5b38ae128a75fe23f58e59619907eb9de1c",
        "domain": "numina",
        "question": "[Example 3.4.6] Let $x_{1}, x_{2}, x_{3}, x_{4}$ be positive real numbers, and $x_{1}+x_{2}+x_{3}+x_{4}=$ $\\pi$. Find the minimum value of the expression\n$$\n\\begin{array}{l}\n\\left(2 \\sin ^{2} x_{1}+\\frac{1}{\\sin ^{2} x_{1}}\\right) \\cdot\\left(2 \\sin ^{2} x_{2}+\\frac{1}{\\sin ^{2} x_{2}}\\right) . \\\\\n\\left(2 \\sin ^{2} x_{3}+\\frac{1}{\\sin ^{2} x_{3}}\\right) \\cdot\\left(2 \\sin ^{2} x_{4}+\\frac{1}{\\sin ^{2} x_{4}}\\right)\n\\end{array}\n$$",
        "answer": "81",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 113839,
        "difficulty": null
      },
      {
        "id": "60d71726720b60e1413797f97ebf2ec7421795034d85a8d34cb631f63e86b19d",
        "domain": "numina",
        "question": "Three of the positive divisors of 24 are 1, 8, and 24 . What is the sum of all of the positive divisors of 24 ?\n\n##",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194483,
        "difficulty": null
      },
      {
        "id": "9f1f4e7cc28e39f6cc43ea4f8dbe60327194ff7e7040610957532b553103e5f4",
        "domain": "numina",
        "question": "[Example 3.5.6] Given that $p$ is a prime number, $r$ is the remainder when $p$ is divided by 210. If $r$ is a composite number and can be expressed as the sum of two perfect squares, find $r$.\n\n保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "169",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 146972,
        "difficulty": null
      },
      {
        "id": "869dcf4a08ba96ef12b5bec12d58276c8408b68d5f0e7104a5540ced4488aa0f",
        "domain": "numina",
        "question": "3. This year, the sum of Piplu and Luluxi's ages is 22 years old, 1 year later Piplu's age will be exactly twice Luluxi's age. This year, Luluxi is $\\qquad$ years old.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 127668,
        "difficulty": null
      },
      {
        "id": "228f82ae52a0423850dce222be57bad4770660350e17406ba8f2338fa9f8d1c9",
        "domain": "numina",
        "question": "$a b c$\nExample 3.1.1 Three-digit addition $\\frac{+d e f}{g h i}$ where $a, b, c, d, e, f, g, h, i$ are exactly composed of 1, $2,3,4,5,6,7,8,9$, find the solutions for the above addition that make $g h i$ the largest and smallest.\n\nAnalysis: One-digit addition has two cases: with carry and without carry. Suppose in the three-digit addition, there are $k$ carries $(k=0,1,2)$. First, explore the necessary conditions that the addition satisfying the conditions should meet.",
        "answer": "981459",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144199,
        "difficulty": null
      },
      {
        "id": "ad97ffb016b115386d01cf1e464a212e3f05d1dca3921ada08dd0705d552b2e7",
        "domain": "numina",
        "question": "## Task $2 / 83$\n\nWe are looking for all natural numbers $n$ for which $3^{n}$, when written in decimal notation, has exactly $0.5 n$ digits.",
        "answer": "42",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182697,
        "difficulty": null
      },
      {
        "id": "586090f441dd7730ace704f028d4398e3a3cc2437018cfe9fc9cb80939154b5a",
        "domain": "numina",
        "question": "Tokarev S.I.\n\nOn the first horizontal line of the chessboard, there are 8 black queens, and on the last - 8 white queens. In what minimum number of moves can the white queens exchange places with the black ones? The white and black queens take turns, moving one queen per move.",
        "answer": "23",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 55720,
        "difficulty": null
      },
      {
        "id": "7f6cde40a5d3bdce27f3776b6dd2bfd7ff30f2c8df5d4eb5fc08d1c02ab2bf91",
        "domain": "numina",
        "question": "[Problems on Mixtures and Concentrations]\n\nFrom two pieces of alloys (with different lead content) weighing 6 and 12 kg, pieces of equal mass were cut. Each of the cut pieces was melted with the remainder of the other piece, after which the percentage of lead in both alloys became the same. What are the masses of each of the cut pieces?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 68942,
        "difficulty": null
      },
      {
        "id": "8f03bad073817ec55323404adf4cacec256ddc3b875cb6e2a061b8253b5820b2",
        "domain": "numina",
        "question": "\\section*{Problem 3}\n\nA circle radius 100 is drawn on squared paper with unit squares. It does not touch any of the grid lines or pass through any of the lattice points. What is the maximum number of squares can it pass through?",
        "answer": "800",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 186962,
        "difficulty": null
      },
      {
        "id": "e5aae18872c83e92f09f2163e28baca82516c56c3654aade3cbe116826e0cc42",
        "domain": "numina",
        "question": "11. A quadrilateral contains an inscribed circle (i.e. a circle tangent to the four sides of the quadrilateral). The ratio of the perimeter of the quadrilateral to that of the circle is $4: 3$. The ratio of the area of the quadrilateral to that of the circle, expressed in its lowest terms, is $a: b$. What is the value of $100 a+49 b$ ?",
        "answer": "547",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156341,
        "difficulty": null
      },
      {
        "id": "7ab69a3b2d91df4b68b8fa9ba1d0fd9dbf42c135e0375c619da437cc0f3d5a83",
        "domain": "numina",
        "question": "3.1. Find the sum\n\n$$\n\\sqrt[7]{(-7)^{7}}+\\sqrt[8]{(-8)^{8}}+\\sqrt[9]{(-9)^{9}}+\\ldots+\\sqrt[100]{(-100)^{100}}\n$$\n\n(Each term is of the form $\\left.\\sqrt[k]{(-k)^{k}}\\right)$",
        "answer": "47",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14139,
        "difficulty": null
      },
      {
        "id": "ee88a97f6618153b016eb55aaf116ab3abaa7dd89b53d02eda6e067f84f93638",
        "domain": "numina",
        "question": "(5) Given the sequence $\\left\\{a_{n}\\right\\}$ with the general term formula $a_{n}=\\frac{1}{(n+1) \\sqrt{n}+n \\sqrt{n+1}}$ $\\left(n \\in \\mathbf{N}^{*}\\right)$, and its first $n$ terms sum as $S_{n}$, then in the sequence $S_{1}, S_{2}, \\cdots, S_{2009}$, the number of rational terms is $\\qquad$ terms.",
        "answer": "43",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141257,
        "difficulty": null
      },
      {
        "id": "8a467248f7a48216cfa1d64462132e099ea63ec7ce2a57b3fa0d604eb1dd944e",
        "domain": "numina",
        "question": "【Question 6】\nThere are two sets of numbers, the sum of the first set of 7 numbers is 84, the average of the second set is 21, and the average of all numbers in both sets is 18, then the second set has\n$\\qquad$ numbers.",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 121589,
        "difficulty": null
      },
      {
        "id": "f90e342c610806bd23c053d9187c122adc132abd3a183af94102e7d2a4a1b76f",
        "domain": "numina",
        "question": "7. Xiao Qiang, while calculating a division problem, wrote the divisor 76 as 67, resulting in a quotient of 15 and a remainder of 5. The correct quotient should be $\\qquad$ _.",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141943,
        "difficulty": null
      },
      {
        "id": "c067588b8efbd7a74a9b7db2341cdadf8386f34a020823c05d981cb62d0a765e",
        "domain": "numina",
        "question": "In the coordinate plane consider the set $S$ of all points with integer coordinates. For a positive integer $k$, two distinct points $A, B \\in S$ will be called $k$-friends if there is a point $C \\in S$ such that the area of the triangle $A B C$ is equal to $k$. A set $T \\subset S$ will be called a $k$-clique if every two points in $T$ are $k$-friends. Find the least positive integer $k$ for which there exists a $k$-clique with more than 200 elements.",
        "answer": "180180",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 196007,
        "difficulty": null
      },
      {
        "id": "8437fd7f8e59b19ace917ac737a443545fbbfebeeae66a2b72dd1bd0d04a3b06",
        "domain": "numina",
        "question": "3.1. Every hour, between two adjacent nettle bushes in a row, two more of the same grow. How many bushes do you need to plant initially so that after three hours, the total number of bushes together is 190?",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13621,
        "difficulty": null
      },
      {
        "id": "5fc5b017c8466794e28a104ca14002119023c30ac5dbcf0397f158d5236526f1",
        "domain": "numina",
        "question": "6. Let $A B C$ be a triangle with incenter $I$, incircle $\\gamma$ and circumcircle $\\Gamma$. Let $M, N, P$ be the midpoints of sides $\\overline{B C}, \\overline{C A}, \\overline{A B}$ and let $E, F$ be the tangency points of $\\gamma$ with $\\overline{C A}$ and $\\overline{A B}$, respectively. Let $U$, $V$ be the intersections of line $E F$ with line $M N$ and line $M P$, respectively, and let $X$ be the midpoint of arc $\\widehat{B A C}$ of $\\Gamma$. Given that $A B=5, A C=8$, and $\\angle A=60^{\\circ}$, compute the area of triangle $X U V$.",
        "answer": "\\frac{21\\sqrt{3}}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151597,
        "difficulty": null
      },
      {
        "id": "599648703b49b51a1043f8e921e86f2b301d7ceeb2c0e63b8e9ce0c42b5b4fe8",
        "domain": "numina",
        "question": "10.2. In the bottom left corner of a $7 \\times 7$ chessboard, there is a king. In one move, he can move one square to the right, or one square up, or one square diagonally to the right and up. In how many different ways can the king travel to the top right corner of the board, if he is forbidden to visit the central cell?",
        "answer": "5020",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18769,
        "difficulty": null
      },
      {
        "id": "da790ceb2533cdf9ed939df213ccdb0f5c382775d28fb6a3a45a59cae01c1b54",
        "domain": "numina",
        "question": "6. To lift a load, it is attached to the hook of a crane using slings made of steel cable. The calculated mass of the load is $M=20$ t, the number of slings $n=3$. Each sling forms an angle $\\alpha=30^{\\circ}$ with the vertical. All slings carry the same load during the lifting of the cargo. According to safety requirements, the tensile strength of the cable should exceed the calculated load by a factor of $k=6$. The cable consists of a very large number of steel threads, each of which can withstand a maximum load of $q=10^{3} \\mathrm{H} / \\mathrm{mm}^{2}$. Indicate the smallest diameter of the cable that can be used to manufacture the slings, if the available cables have any integer diameter in millimeters. The acceleration due to gravity $g$ is considered to be $10 \\mathrm{~m} / \\mathrm{s}^{2}$.",
        "answer": "26",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11161,
        "difficulty": null
      },
      {
        "id": "dc9a5ab53ccf20a1ba738c452a05956fdaf478a12c4ce711e066fbcc5855de69",
        "domain": "numina",
        "question": "3. Given $\\angle A, \\angle B, \\angle C$ are the three interior angles of $\\triangle ABC$, and the vector $\\alpha=\\left(\\cos \\frac{A-B}{2}, \\sqrt{3} \\sin \\frac{A+B}{2}\\right)$, $|\\alpha|=\\sqrt{2}$. If when $\\angle C$ is maximized, there exists a moving point $M$ such that $|M A|, |A B|, |M B|$ form an arithmetic sequence, the maximum value of $|A B|$ is $\\qquad$",
        "answer": "\\frac{2\\sqrt{3}+\\sqrt{2}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 146553,
        "difficulty": null
      },
      {
        "id": "0d8927885072b56e550c7f8b2ffb622244a980cc5f3841ead0fe59745f521dc9",
        "domain": "numina",
        "question": "2. (10 points) The little rabbit and the little turtle start from location $A$ to the forest amusement park at the same time. The little rabbit jumps forward 36 meters per minute, and after every 3 minutes of jumping, it plays on the spot. The first time it plays for 0.5 minutes, the second time for 1 minute, the third time for 1.5 minutes, $\\cdots$, the $k$-th time for $0.5 k$ minutes. The little turtle does not rest or play on the way. It is known that the little turtle arrives at the forest amusement park 3 minutes and 20 seconds earlier than the little rabbit. The distance from $A$ to the forest amusement park is 2640 meters. Then, the little turtle crawls $\\qquad$ meters per minute.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136381,
        "difficulty": null
      },
      {
        "id": "d54015dce66769b2565a0407ef32116b6351b4294f339444564e02ae34a19ee5",
        "domain": "numina",
        "question": "2.15. Find the lateral surface of a regular triangular prism with height $h$, if a line passing through the center of the upper base and the midpoint of the side of the lower base is inclined to the plane of the base at an angle of $60^{\\circ}$.",
        "answer": "6^{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 50114,
        "difficulty": null
      },
      {
        "id": "3e37d556fc0af374d5fcceb9595c81a5eeea52395f95483a53f92c7917720e04",
        "domain": "numina",
        "question": "5. Triangle $A B C$ has side lengths $A B=2 \\sqrt{5}, B C=1$, and $C A=5$. Point $D$ is on side $A C$ such that $C D=1$, and $F$ is a point such that $B F=2$ and $C F=3$. Let $E$ be the intersection of lines $A B$ and $D F$. Find the area of $C D E B$.",
        "answer": "\\frac{22}{35}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152409,
        "difficulty": null
      },
      {
        "id": "e7f64bb37dd1d76e9947feb2a8636586fd0b5fb8e31510faa8187e6bf4fb95d9",
        "domain": "numina",
        "question": "18. (3 points) When Xiaoming goes from home to school, he walks the first half of the distance and takes a ride for the second half; when he returns from school to home, he rides for the first $\\frac{1}{3}$ of the time and walks for the last $\\frac{2}{3}$ of the time. As a result, the time it takes to go to school is 2 hours more than the time it takes to return home. It is known that Xiaoming walks at a speed of 5 kilometers per hour and rides at a speed of 15 kilometers per hour. Therefore, the distance from Xiaoming's home to school is $\\qquad$ kilometers.",
        "answer": "150",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143989,
        "difficulty": null
      },
      {
        "id": "26907f9880a49bc7922e1685df78cfb144670a0524c6307b6829522982d97417",
        "domain": "numina",
        "question": "410. The distribution function of a two-dimensional random variable is given by\n\n$$\nF(x, y)=\\left\\{\\begin{array}{c}\n\\sin x \\cdot \\sin y \\text { for } 0 \\leqslant x \\leqslant \\pi / 2,0 \\leqslant y \\leqslant \\pi / 2 \\\\\n0 \\quad \\text { for } x<0 \\text { or } y<0\n\\end{array}\\right.\n$$\n\nFind the probability that a random point $(X, Y)$ falls within the rectangle bounded by the lines $x=0, x=\\pi / 4$, $\\boldsymbol{y}=\\pi / 6, y=\\pi / 3$.",
        "answer": "\\frac{(\\sqrt{6}-\\sqrt{2})}{4}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32423,
        "difficulty": null
      },
      {
        "id": "77667b4b9e3add7625822cd0836e362a281727dd4a9cdd35291de2b2dfb3c3f0",
        "domain": "numina",
        "question": "12. Given an integer $k$ satisfying $1819<k<2019$, and making the solution of the system of linear equations in two variables $\\left\\{\\begin{array}{l}4 x-5 y=6 \\\\ 5 x+4 y=k\\end{array}\\right.$\n\nall integers, then the number of $k$ that satisfies the condition is $\\qquad$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139961,
        "difficulty": null
      },
      {
        "id": "59a27b1f88ee2d86dd6e72f7dc2d9abc081ede3963347b5409d4825fdedac017",
        "domain": "numina",
        "question": "8. (10 points) In the expression $(x+y+z)^{2020}+(x-y-z)^{2020}$, the brackets were expanded and like terms were combined. How many monomials $x^{a} y^{b} z^{c}$ with a non-zero coefficient were obtained?",
        "answer": "1022121",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19513,
        "difficulty": null
      },
      {
        "id": "e542e5708a1d56a60b7e4f8c6c0393c005a51e23c2a2b9052a44f70af74b4178",
        "domain": "numina",
        "question": "7.091. $\\log _{2}\\left(4 \\cdot 3^{x}-6\\right)-\\log _{2}\\left(9^{x}-6\\right)=1$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50857,
        "difficulty": null
      },
      {
        "id": "24c597bdda7abd278f7f13e3ca7870bc35f3dc0c28cc8bd6dc65ee3f739085a1",
        "domain": "numina",
        "question": "1 A square grid of 16 dots (see the figure) contains the corners of nine $1 \\times 1$ squares, four $2 \\times 2$ squares, and one $3 \\times 3$ square, for a total of 14 squares whose sides are parallel to the sides of the grid. What is the smallest possible number of dots you can remove so that, after removing those dots, each of the 14 squares is missing at least one corner?\nJustify your answer by showing both that the number of dots you claim is sufficient and by explaining why no smaller number of dots will work.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132196,
        "difficulty": null
      },
      {
        "id": "6c49c41c7f0c9f960ea969ec6e400fa2ffc5053de3e70d6a39fb9b210a7ab1b8",
        "domain": "numina",
        "question": "4. Let $A, B, C$ be the centers of the given circles, $A B=2, A C=2, B C=3$ (see Fig. 1).\n\nDenote by $a, b$, and $c$ the radii of the circles centered at points $A, B$, and $C$ respectively, and find their values by solving the system of equations. We get:\n\n$$\n\\left\\{\\begin{array}{l}\na + b = 2 \\\\\na + c = 2 \\\\\nb + c = 3\n\\end{array} \\Leftrightarrow \\left\\{\\begin{array}{l}\na=\\frac{1}{2} \\\\\nb=\\frac{3}{2} \\\\\nc=\\frac{3}{2}\n\\end{array}\\right.\\right.\n$$\n\nLet $S$ and $L$ be the centers of the small and large circles, respectively.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_56ffd9f61e4fedda1946g-2.jpg?height=617&width=625&top_left_y=157&top_left_x=1178)\n\nFig. 1\n\nDenote by $r$ and $R$ the radii of the small and large circles that are tangent to the three given circles, respectively.\n\nLet $\\angle A S B=2 \\gamma, \\angle A S C=2 \\gamma$, and $\\angle B S C=2 \\alpha$. Then, by the cosine rule for triangle $A S B$, we get:\n\n$$\n\\cos 2 \\gamma=\\frac{(a+r)^{2}+(b+r)^{2}-(a+b)^{2}}{2(a+r)(b+r)}=\\frac{r^{2}+(a+b) r-a b}{(a+r)(b+r)}\n$$\n\nFrom this, using the double-angle formulas, we find:\n\n$$\n\\cos ^{2} \\gamma=\\frac{r(a+b+r)}{(a+r)(b+r)}, \\sin ^{2} \\gamma=\\frac{a b}{(a+r)(b+r)}\n$$\n\nSimilarly, for triangle $B S C$, we get:\n\n$$\n\\sin ^{2} \\alpha=\\frac{b^{2}}{(b+r)^{2}}\n$$\n\nSince $\\alpha+2 \\gamma=\\pi$, then $\\sin ^{2} \\alpha=\\sin ^{2} 2 \\gamma=4 \\sin ^{2} \\gamma \\cos ^{2} \\gamma$.\n\nFrom this, we have:\n\n$$\n\\begin{gathered}\n\\frac{b^{2}}{(b+r)^{2}}=4 \\cdot \\frac{a b}{(a+r)(b+r)} \\cdot \\frac{r(a+b+r)}{(a+r)(b+r)} \\\\\nb^{2}(a+r)^{2}=4 a b(a+b+r) \\\\\n(b-4 a) r^{2}+\\left(2 a b^{2}-4 a^{2}-4 a b\\right) r+a^{2} b=0\n\\end{gathered}\n$$\n\nSubstituting the found values of the radii of the original circles, we get the equation for determining the value of $r$:\n\n$$\n\\begin{aligned}\n& r^{2}+5 r-\\frac{3}{4}=0 \\\\\n& r=\\frac{1}{2}(2 \\sqrt{7}-5)\n\\end{aligned}\n$$\n\nNote that to find the value of the radius of the large circle $R$, in all the equations for finding the radius of the small circle, we can replace $r$ with $-R$. Then we get the equation:\n\n$$\n\\begin{aligned}\n& R^{2}-5 R-\\frac{3}{4}=0 \\\\\n& R=\\frac{1}{2}(2 \\sqrt{7}+5)\n\\end{aligned}\n$$\n\nWe find the area of the desired figure, we get: $S=\\pi R^{2}-\\pi r^{2}=10 \\sqrt{7} \\pi$.\n\nNote. The radii of the original circles are found - 1 point. The radius of one of the circles is found - 4 points.",
        "answer": "10\\sqrt{7}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15646,
        "difficulty": null
      },
      {
        "id": "77c67fc5680ca0dd7e4c96ba8312af3435331630ae7d6bfaece8be68319881e5",
        "domain": "numina",
        "question": "Example 8 As shown in Figure 1.4.9, in trapezoid $A B C D$, $A B / / D C, C E$ is the bisector of $\\angle B C D$, $C E \\perp A D$ at $E, D E=2 A E, C E$ divides the trapezoid into two parts with areas $S_{1}$ and $S_{2}$, if $S_{1}=1$, find $S_{2}$.",
        "answer": "\\frac{7}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137498,
        "difficulty": null
      },
      {
        "id": "e16bcdbaf12078ac6f66d98786573a5f7e039b04c2d26ca031292551524f5b20",
        "domain": "numina",
        "question": "11. (12 points) There is a new type of problem-solving robot that can solve problems, but it has an IQ balance limit. Each time it solves a problem, it subtracts the score of the problem from its IQ balance, consuming an amount of IQ balance equal to the score. When it solves a problem correctly, its IQ balance increases by 1. If its IQ balance is less than the score of the problem it is solving, it will fail to solve the problem. If Xiaopeng uses a problem-solving robot with an initial IQ limit of 25 to solve a set of problems with scores ranging from $1-10$, what is the maximum score he can achieve? $\\qquad$",
        "answer": "31",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 121278,
        "difficulty": null
      },
      {
        "id": "24252454cd655dd3ef3a7f7f99afdda7c665265f9b56c12ab6fe22d99d8ae413",
        "domain": "numina",
        "question": "2. For a rectangle, the sum of two sides is 11, and the sum of three sides is 19.5. Find the product of all possible different values of the perimeter of such a rectangle.",
        "answer": "15400",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10255,
        "difficulty": null
      },
      {
        "id": "1c7b9d8a0260e17ad4a5ef3ae54e49dde8f8f1a4490f3169d0134c53a1a54dde",
        "domain": "numina",
        "question": "2. (2004 College Entrance Examination - Zhejiang Paper) In $\\triangle A B C$, the sides opposite to angles $A, B, C$ are $a, b, c$ respectively, and $\\cos A=\\frac{1}{3}$.\n(1) Find the value of $\\sin ^{2} \\frac{B+C}{2}+\\cos 2 A$;\n(2) If $a=\\sqrt{3}$, find the maximum value of $b c$.\n\n保留了原文的换行和格式，如上所示。",
        "answer": "\\frac{9}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123266,
        "difficulty": null
      },
      {
        "id": "d0cd3bd6ea351df7ba72ef2d464637971c98ffb8ef0cd7d3c096eb145bc53ed3",
        "domain": "numina",
        "question": "Problem 6. A row of new recruits stood facing the sergeant. On the command \"left,\" some turned left, while the rest turned right. It turned out that six times more soldiers were looking at the back of their neighbor than in the face. Then, on the command \"about face,\" everyone turned in the opposite direction. Now, seven times more soldiers were looking at the back of their neighbor than in the face. How many soldiers are in the row?\n\n$[8$ points] (A. V. Shipovalov)",
        "answer": "98",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22746,
        "difficulty": null
      },
      {
        "id": "06c6dd0f65dd60210cc6b591e73d4ff701f4859de309f36b4d5abed1c43b24da",
        "domain": "numina",
        "question": "18. (6 points) In a certain exam, the average score of 11 students, rounded to the first decimal place, is 85.3. It is known that each student's score is an integer. Therefore, the total score of these 11 students is $\\qquad$ points.",
        "answer": "938",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 138218,
        "difficulty": null
      },
      {
        "id": "4981654427a5592ee76b9464a68255e90a6bba139762a0e8760e97379f1ef56b",
        "domain": "numina",
        "question": "2. $x, y \\in \\mathbf{R}$, and satisfy $\\left\\{\\begin{array}{l}(x+1)^{\\frac{3}{5}}+2023(x+1)=-2023, \\\\ (y+1)^{\\frac{3}{5}}+2023(y+1)=2023\\end{array}\\right.$, then $x+y=$",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113722,
        "difficulty": null
      },
      {
        "id": "dc54ab36c76851447538c75798c2bbd438f7416fbfa01a055e922d370c529127",
        "domain": "numina",
        "question": "2. In $\\triangle A B C$, $M, N$ are points on line segments $A B, A C$ respectively, and $A M=\\frac{1}{2} A B, A N=\\frac{1}{3} A C, E, F, G$ are three moving points on line segment $B C$, and $\\overrightarrow{A E}+\\overrightarrow{A F}+\\overrightarrow{A G}=x \\overrightarrow{A M}+y \\overrightarrow{A N}(x, y \\in \\mathbf{R})$, then the minimum value of $\\frac{2}{x}+\\frac{3}{y}$ is $\\qquad$",
        "answer": "\\frac{4}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124111,
        "difficulty": null
      },
      {
        "id": "be6d7b61b276a41c1bc3f9f92696a104fef120fdfd2c340731987aa083fdce0e",
        "domain": "numina",
        "question": "36. [25] A prime number $p$ is twin if at least one of $p+2$ or $p-2$ is prime and sexy if at least one of $p+6$ and $p-6$ is prime.\nHow many sexy twin primes (i.e. primes that are both twin and sexy) are there less than $10^{9}$ ? Express your answer as a positive integer $N$ in decimal notation; for example, 521495223. If your answer is in this form, your score for this problem will be $\\max \\left\\{0,25-\\left\\lfloor\\frac{1}{10000}|A-N|\\right\\rfloor\\right\\}$, where $A$ is the actual answer to this problem. Otherwise, your score will be zero.",
        "answer": "1462105",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151567,
        "difficulty": null
      },
      {
        "id": "8cf234f8b9c502c96b55891e67179f953aa1a9b4c8c08f8d1895f21b47309090",
        "domain": "numina",
        "question": "Problem 6. Plot the figure on the coordinate plane defined by the system of inequalities and find its area $\\left\\{\\begin{array}{c}|x+5|+\\sqrt{3}|y-1| \\leq 3 \\\\ y \\leq \\sqrt{4-4 x-x^{2}}+1 \\\\ |2 y-1| \\leq 5\\end{array}\\right.$.",
        "answer": "\\frac{2}{3}\\pi+\\frac{4}{3}\\sqrt{3}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 10555,
        "difficulty": null
      },
      {
        "id": "77794dcaa79387c369b9309c2e920007ba0c466bb86d0b45b3fe3b9657bde3b8",
        "domain": "numina",
        "question": "16. [10] If $p$ and $q$ are positive integers and $\\frac{2008}{2009}<\\frac{p}{q}<\\frac{2009}{2010}$, what is the minimum value of $p$ ?",
        "answer": "4017",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151046,
        "difficulty": null
      },
      {
        "id": "8599f0079cfe4cca7b4ff3e3f598a48c71b848a891fbdfd17c4a52f5bee0b31e",
        "domain": "numina",
        "question": "9. From the 10 numbers $0,1,2,3,4,5,6,7,8,9$, choose 3 numbers such that their sum is an even number not less than 10. The number of different ways to do this is $\\qquad$.",
        "answer": "51",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125582,
        "difficulty": null
      },
      {
        "id": "68ad6cf2392132ce3677bc2028b011c367b289a92567c5be74f403581fc3481b",
        "domain": "numina",
        "question": "## Task 1 - 090931\n\nLet $A B C D E F G H$ be a regular octagon. Consider all triangles formed by any three of the points $A, B, C, D, E, F, G, H$.\n\nSomeone wants to list some of these triangles, with the condition that no two of the listed triangles are congruent to each other.\n\nDetermine the maximum number of triangles he can list under this condition!",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 184476,
        "difficulty": null
      },
      {
        "id": "f24939c1655235dce0a6e6c67ed8cee320b157510e82c5db279dbc67179d9aa2",
        "domain": "numina",
        "question": "## Task B-1.2.\n\nGiven the expression $\\quad|x-1|+|x-2|+|x-3|+\\ldots+|x-2019|+|x-2020|$.\n\nDetermine the largest interval of real numbers $[a, b]$ on which the given expression has a constant value $k$ for all $x \\in[a, b]$. What is the value of the constant $k$?",
        "answer": "1010^2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87861,
        "difficulty": null
      },
      {
        "id": "7bd2e88975693efd0c91eb1773c1b8f4d3dc1da841d93175736fb84b0c89a2f7",
        "domain": "numina",
        "question": "7. In a trapezoid, the diagonals of which intersect at a right angle, it is known that the midline is 6.5 and one of the diagonals is 12. Find the second diagonal\n\n#",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5329,
        "difficulty": null
      },
      {
        "id": "f59cdcaca335646a0b53cfc3b54cc1eb7559a170cfda98afd152e32197669fe2",
        "domain": "numina",
        "question": "10. (12 points) The figure is composed of 5 identical squares. The number of triangles that can be formed using the 12 points in the figure as vertices is.\n\n保留了原文的换行和格式，以下是翻译结果：\n```\n10. (12 points) The figure is composed of 5 identical squares. The number of triangles that can be formed using the 12 points in the figure as vertices is.\n```",
        "answer": "200",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 108933,
        "difficulty": null
      },
      {
        "id": "a6ea1a1ad93507e1682813af63be3c713295742469dc19ffa44fe75003ad3695",
        "domain": "numina",
        "question": "In the box, there are blue, red, and green pencils. In total, there are 20 pieces. Blue pencils are 6 times more than green ones, and there are fewer red pencils than blue ones.\n\nHow many red pencils are in the box?\n\n#",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 71453,
        "difficulty": null
      },
      {
        "id": "840bd6046e3160a5a8bb21252259d5dc654de1b073b4ed084778d04dd30229d9",
        "domain": "numina",
        "question": "5. On the cells of a chessboard of size $8 \\times 8$, 6 identical pieces are randomly placed. Find the probability that five of them will be located either on the same row, or on the same column, or on one of the two main diagonals.",
        "answer": "\\frac{339}{446276}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 2272,
        "difficulty": null
      },
      {
        "id": "7e1617c4e79e5fb46ff3d0e725402b8d1c73e0eb1a5651716f538a5b1cd49bd2",
        "domain": "numina",
        "question": "Trushin 5.\n100 thimbles are arranged in a circle. Under one of them, a coin is hidden. In one move, it is allowed to flip over four thimbles and check if the coin is under one of them. After this, they are returned to their original position, and the coin is moved to one of the adjacent thimbles. What is the minimum number of moves required to definitely find the coin?",
        "answer": "33",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 55479,
        "difficulty": null
      },
      {
        "id": "a415e7c943fa9e40dad90755845a9a8fae320388a046f47fded46471e61312d2",
        "domain": "numina",
        "question": "G4.2 Suppose $P(x)=a_{0}+a_{1} x+a_{2} x^{2}+\\ldots+a_{8} x^{8}$ is a polynomial of degree 8 with real coefficients $a_{0}, a_{1}, \\ldots, a_{8}$. If $P(k)=\\frac{1}{k}$ when $k=1,2, \\ldots, 9$, and $b=P(10)$, find the value of $b$.",
        "answer": "\\frac{1}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152983,
        "difficulty": null
      },
      {
        "id": "15701253315796b3117523514e7a4ead935629b36fcf2890d8bd18eab03f004e",
        "domain": "numina",
        "question": "Problem 8.4. In a class, there are 20 students. Thinking about which girls to send a Valentine's card to on February 14, each boy made a list of all the girls in the class he finds attractive (possibly an empty list). It is known that there do not exist three boys whose lists have the same number of girls. What is the minimum number of girls that can be in the class?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12892,
        "difficulty": null
      },
      {
        "id": "a1288b141f35d620ed7aa55d8e138be2d783ab2bd434e6d6b5bd940821ed766f",
        "domain": "numina",
        "question": "5. All edges of a regular quadrilateral pyramid $S A B C D$ with base $A B C D$ have equal length. Plane $\\alpha$ is perpendicular to line $S A$, and plane $\\beta$ is parallel to line $C D$. Determine the smallest possible angle between planes $\\alpha$ and $\\beta$. (7 points) Answer: $\\frac{\\pi}{6}$.",
        "answer": "\\frac{\\pi}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8539,
        "difficulty": null
      },
      {
        "id": "5d5a8c11dd2f9b2588d711310851bcf760278aea9423d2d9d34c1389b652933f",
        "domain": "numina",
        "question": "1. (8 points) The calculation result of the expression $(19 \\times 19-12 \\times 12) \\div\\left(\\frac{19}{12}-\\frac{12}{19}\\right)$ is",
        "answer": "228",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115754,
        "difficulty": null
      },
      {
        "id": "50ccbcc064bdc9608025791f57c1f4cfe172509e31e6269a1dfef7f09d0bf75e",
        "domain": "numina",
        "question": "5. Given the sequence $\\left\\{a_{n}\\right\\}$, the first two terms are $a_{1}=2, a_{2}=4$. Let $b_{n}=a_{n+1}-a_{n}$. If the sequence $\\left\\{b_{n}\\right\\}$ is an arithmetic sequence with a common difference of 1, then $a_{2020}=$ $\\qquad$",
        "answer": "2041211",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96466,
        "difficulty": null
      },
      {
        "id": "1aa70868e2e4d7d6ba58cae457c227bef809c73a25c019c726fdbae97e15a970",
        "domain": "numina",
        "question": "What is the smallest two-digit positive integer that is a multiple of 6 but is not a multiple of 4 ?",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194363,
        "difficulty": null
      },
      {
        "id": "0b5f941560e190ca6d604d780671c47487e1c7096c546d9e1d03ff143972e83e",
        "domain": "numina",
        "question": "Task 15. (8 points)\n\nAnnouncement: \"Have free time and want to earn money? Write now and earn up to 2500 rubles a day working as a courier with the service 'Food.There-Here!'. Delivery of food from stores, cafes, and restaurants.\n\nHow much will a school student working as a courier with the service 'Food.There-Here!' earn in a month (4 weeks), working 4 days a week with a minimum load for 1250 rubles a day after the income tax deduction?\n\nIn your answer, provide only the number without units of measurement! #",
        "answer": "17400",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11477,
        "difficulty": null
      },
      {
        "id": "a7655562bd3fc59c5b9f798c92d66a7b57f6a0367af9342f24606b9459719040",
        "domain": "numina",
        "question": "Question 215, Let the sum of all positive divisors of the positive integer $\\mathrm{n}$ be $\\sigma(\\mathrm{n})$, find the value: $\\left[\\sqrt[3]{\\sum_{\\mathrm{n}=1}^{2020} \\frac{\\sigma(\\mathrm{n}}{\\mathrm{n}}}\\right]$, here $[\\mathrm{x}]$ represents the greatest integer not exceeding $x$.\n\n保留源文本的换行和格式，翻译结果如下：\n\nQuestion 215, Let the sum of all positive divisors of the positive integer $\\mathrm{n}$ be $\\sigma(\\mathrm{n})$, find the value: $\\left[\\sqrt[3]{\\sum_{\\mathrm{n}=1}^{2020} \\frac{\\sigma(\\mathrm{n}}{\\mathrm{n}}}\\right]$, here $[\\mathrm{x}]$ represents the greatest integer not exceeding $x$.",
        "answer": "14",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 145022,
        "difficulty": null
      },
      {
        "id": "b79793eff931702e6a5e712e92fb1642e1c3ade0e6c1c3a0b88ffcf4063eb998",
        "domain": "numina",
        "question": "3. Translator (from 6th grade. 1 point). Vasily Petrov is completing an English language assignment. In this assignment, there are 10 English expressions and their Russian translations in random order. The task is to establish the correct correspondences between the expressions and their translations. 1 point is given for each correctly established correspondence. Thus, one can score from 0 to 10 points. Vasya knows nothing, so he chooses the options at random. Find the probability that he will score exactly 9 points.",
        "answer": "0",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10952,
        "difficulty": null
      },
      {
        "id": "0f3244313b92e38cd2a7dfc0379eb571d9c1f1e3f3e8a466d68b8ec023dfc052",
        "domain": "numina",
        "question": "1. Given a finite set of cards. On each of them, either the number 1 or the number -1 is written (exactly one number on each card), and there are 100 more cards with -1 than cards with 1. If for each pair of different cards, the product of the numbers on them is found, and all these products are summed, the result is 1000. How many cards have the number 1?",
        "answer": "3950",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12275,
        "difficulty": null
      },
      {
        "id": "fbabe13810f3bad4b5cf40cff150ff9c4075bd0b0e7564ba186ab06a9c0c8693",
        "domain": "numina",
        "question": "19. Given 7 points on a circle, connect each pair of points with line segments. What is the minimum number of triangles in the resulting figure?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "284",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127944,
        "difficulty": null
      },
      {
        "id": "53740138daf299484a1b847b634acb5fa9554f01b68b6603c0f16d9a38c49d40",
        "domain": "numina",
        "question": "Task 1. Find the largest four-digit number in which all digits are different, and moreover, no two of them can be swapped to form a smaller number.",
        "answer": "7089",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6377,
        "difficulty": null
      },
      {
        "id": "28912ae3a69acac934932fb3c5f74876b1ba6385dc890870a852b0d1abfae652",
        "domain": "numina",
        "question": "Let $A_{1} A_{2} A_{3}$ be a triangle. We construct the following points:\n\n- $B_{1}, B_{2}$, and $B_{3}$ are the midpoints of $A_{1} A_{2}, A_{2} A_{3}$, and $A_{3} A_{1}$.\n\n- $C_{1}, C_{2}$, and $C_{3}$ are the midpoints of $A_{1} B_{1}, A_{2} B_{2}$, and $A_{3} B_{3}$.\n\n- $D_{1}$ is the intersection of $\\left(A_{1} C_{2}\\right)$ and $\\left(B_{1} A_{3}\\right)$, $D_{2}$ and $D_{3}$ are defined cyclically in the same way.\n- $E_{1}$ is the intersection of $\\left(A_{1} B_{2}\\right)$ and $\\left(C_{1} A_{3}\\right)$, $E_{2}$ and $E_{3}$ are defined cyclically in the same way.\n\nCalculate the quotient of the area of $\\mathrm{D}_{1} \\mathrm{D}_{2} \\mathrm{D}_{3}$ by the area of $\\mathrm{E}_{1} \\mathrm{E}_{2} \\mathrm{E}_{3}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_10_59677aa68548da326d08g-214.jpg?height=632&width=1371&top_left_y=1465&top_left_x=366)\n\n## 2 Solutions",
        "answer": "\\frac{25}{49}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 80976,
        "difficulty": null
      },
      {
        "id": "3a581ce637be6488dcc04ef6a3504b8ca5023dd97501022465111ee856d57dd2",
        "domain": "numina",
        "question": "13.9 Given several sets with identical contents, each set consists of 4 objects with integer gram weights. Now, we want to use these objects as weights to measure every integer gram weight from 1 gram to 1985 grams (including 1985 grams). To minimize the total weight of the objects used as weights, how many different ways are there to select the weights of the 4 objects?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112087,
        "difficulty": null
      },
      {
        "id": "75ffd0a080f511ab5b07f56cd684742b271295f4b00b0d7807d94d6a4971a3ec",
        "domain": "numina",
        "question": "32. How many five-digit numbers are there that read the same from left to right and right to left?",
        "answer": "900",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44853,
        "difficulty": null
      },
      {
        "id": "8f6a30ced77c089a74483a6018d8d96591d50142988cab783cbc30f8454a96af",
        "domain": "numina",
        "question": "75. In quadrilateral $A B C D$, it is given that $\\widehat{D A B}=150^{\\circ}$, $\\widehat{D A C}+\\widehat{A B D}=120^{\\circ}, \\widehat{D B C}-\\widehat{A B \\bar{D}}=60^{\\circ}$. Find $\\widehat{B D C}$.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 24883,
        "difficulty": null
      },
      {
        "id": "6ae5d4034bbbc7bf4c081343c3e8f311000181ff8aee45ed06da69f2b4eb8570",
        "domain": "numina",
        "question": "A four-digit number, which is a perfect square of a number, has its first two and last two digits equal; find this number.",
        "answer": "7744",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 172552,
        "difficulty": null
      },
      {
        "id": "b34963dc782c6ee3636878bd0724b9e8ae7111cc71918cf3dea54ef5bb452ee0",
        "domain": "numina",
        "question": "Someone said in the summer: \"My birth year is divisible by my current age.\" A friend replied: \"Although I am older than you, I can still say the same about myself.\" - How old are the speakers?",
        "answer": "3753",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 174892,
        "difficulty": null
      },
      {
        "id": "83b584185aad3ff0eabae552462bc8c63cbe93418225beca944539ac38471547",
        "domain": "numina",
        "question": "【Question 13】\nIn a city's archery competition, after ranking the scores, the average score of the top seven is 3 points less than the average score of the top four, and the average score of the top ten is 4 points less than the average score of the top seven. The total score of the 5th, 6th, and 7th place is $\\qquad$ points more than the total score of the 8th, 9th, and 10th place.",
        "answer": "28",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114950,
        "difficulty": null
      },
      {
        "id": "520dc56fbeea8f798e7c068fb2b33946114f4e065f47513e6cf564fbdde566f5",
        "domain": "numina",
        "question": "1. Calculate: $\\frac{9 \\times 11+31}{2015} \\times \\frac{31+6 \\frac{1}{5}+4 \\frac{3}{7}}{1+\\frac{1}{5}+\\frac{1}{7}}=$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103682,
        "difficulty": null
      },
      {
        "id": "e04934f54ba4b67eb2c45f2391572fd94bc4c5ba0a7615bad5cd462168d03082",
        "domain": "numina",
        "question": "(4) There are three sets of cards, red, yellow, and blue, each set containing five cards, marked with the letters $A, B, C, D$, and $E$. If 5 cards are drawn from these 15 cards, with the requirement that the letters are all different and all three colors are included, then the number of different ways to draw the cards is $\\qquad$ kinds.",
        "answer": "150",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 105611,
        "difficulty": null
      },
      {
        "id": "9d7d888bbe64b6adb09b48629f457111697e9e64fd91dbcbf496ac73e37d2029",
        "domain": "numina",
        "question": "Example 16 (28th Brazilian Mathematical Olympiad) If a positive integer has 8 positive divisors, and the sum of these 8 positive divisors is 3240, then this positive integer is called a \"good number\". For example, 2006 is a good number, because the sum of its divisors 1, 2, 17, 34, 59, 118, 1003, 2006 is 3240. Find the smallest good number.",
        "answer": "1614",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122053,
        "difficulty": null
      },
      {
        "id": "f279f69ae5f29cdd82be607364fa5f6d96c8eebebd12c29cbde376ac4d203ab8",
        "domain": "numina",
        "question": "12. The teacher is buying souvenirs for the students. There are three different types of souvenirs in the store (souvenirs of the same type are identical), priced at 1 yuan, 2 yuan, and 4 yuan respectively. The teacher plans to spend 101 yuan, and at least one of each type of souvenir must be purchased. There are $\\qquad$ different purchasing schemes.",
        "answer": "600",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104434,
        "difficulty": null
      },
      {
        "id": "f304ce71f04d2b2d928ca0a8861cfd3d6d13b944754acdd60370f35308f564c1",
        "domain": "numina",
        "question": "$4 \\cdot 41$ Solve the equation $5 x^{2}+x-x \\cdot \\sqrt{5 x^{2}-1}-2=0$.",
        "answer": "\\\\frac{\\sqrt{10}}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103972,
        "difficulty": null
      },
      {
        "id": "bee2c0d67f0202f85e917f1c4a16ff65be98b1c199cae86a4ac42d37ecec2ee6",
        "domain": "numina",
        "question": "11.2. How much greater is one of two positive numbers than the other if their arithmetic mean is $2 \\sqrt{3}$ and the geometric mean is $\\sqrt{3}$?\n\n(Hint: the geometric mean of two numbers $m$ and $n$ is the number $p=\\sqrt{\\mathrm{mn}}$).",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13923,
        "difficulty": null
      },
      {
        "id": "39195513aa46c00323fb88189ba188da026e1f5f716209392924bbbabca71e7e",
        "domain": "numina",
        "question": "1. Calculate the remainder obtained when dividing $2^{4 n+2}+2^{4}$ by $2^{2 n+1}+2^{n+1}+1$ for every natural number $n \\geq 2$.",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 187984,
        "difficulty": null
      },
      {
        "id": "54f70e38d416d79e34a67a62e198274089ca54c6db163bf0882946295048896a",
        "domain": "numina",
        "question": "## 212. Math Puzzler $1 / 83$\n\nIn a shoe factory, 100 workers produce 500 pairs of shoes daily. During a Subbotnik, this daily performance is exceeded by 10 percent of the workers by 10 percent.\n\nWhat percentage did the company bill at the end of the voluntary work effort? How many pairs of shoes were produced? What did the production director need to pay attention to?",
        "answer": "101",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 181883,
        "difficulty": null
      },
      {
        "id": "7ed6d1835bef8a189fc2fe8a2ce52fea6c15661ac35ab5dca6999fcb58ee8b0f",
        "domain": "numina",
        "question": "Problem 2. Along the path between the houses of Nезнайка (Nезнayka) and Синеглазка (Sineglazka), there were 15 peonies and 15 tulips growing in a row, mixed together.\n\nSetting out from home to visit Nезнайка, Синеглазка watered all the flowers in a row. After the 10th tulip, the water ran out, and 10 flowers remained unwatered.\n\nThe next day, setting out from home to visit Синеглазка, Незнайка picked flowers for her one by one. After picking the 6th tulip, he decided that it was enough for the bouquet. How many flowers remained growing along the path?",
        "answer": "19",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22934,
        "difficulty": null
      },
      {
        "id": "ea74ecf76b698df4cce8321b0f10bad1e75a91caabd6bdbfa8293984f40ee8dc",
        "domain": "numina",
        "question": "## SUBJECT 4\n\nLet the sequence of real numbers $\\left(x_{n}\\right)_{n \\geq 1}$, given by $x_{1}=\\alpha$, with $\\alpha \\in \\mathrm{R}^{*}, \\alpha \\neq 2$ and $x_{n+1}=\\frac{x_{n}^{2}+2 x_{n}+2}{x_{n}^{2}+1},(\\forall) n \\geq 1$. Calculate:\na) $\\lim _{n \\rightarrow \\infty} x_{n}$,\nb) $\\lim _{n \\rightarrow \\infty}\\left(x_{n}-1\\right)^{n}$.\n\nProf. Dorin Arventiev\n\n## Note:\n\nWorking time: 3 hours.\n\nAll problems are mandatory.\n\nEach problem is graded from 0 to 7.\n\nNo points are given ex officio.\n\n## MATHEMATICS OLYMPIAD\n\nLocal stage - Constanța, 21.02.2016\n\n11th Grade\n\n## Grading and marking rubric\n\nNo points are given ex officio.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178276,
        "difficulty": null
      },
      {
        "id": "40198b500b78c9787394649ec2f732d0a89ef04fcf7f11746ca5178055f2c982",
        "domain": "numina",
        "question": "6. Given that $P$ is a point on the hyperbola $C: \\frac{x^{2}}{4}-\\frac{y^{2}}{12}=1$, $F_{1}$ and $F_{2}$ are the left and right foci of the hyperbola $C$, and $M$ and $I$ are the centroid and incenter of $\\triangle P F_{1} F_{2}$, respectively. If $M I \\perp x$-axis, then the radius of the incircle of $\\triangle P F_{1} F_{2}$ is $\\qquad$",
        "answer": "\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97176,
        "difficulty": null
      },
      {
        "id": "e377b66a0bd1b32af6a2eae944c637b2878c0f138d6d007f703905fea469d2ac",
        "domain": "numina",
        "question": "Problem 2. On the side $AB$ of triangle $ABC$, a point $M$ is taken. It starts moving parallel to $BC$ until it intersects with $AC$, then it moves parallel to $AB$ until it intersects with $BC$, and so on. Is it true that after a certain number of such steps, point $M$ will return to its original position? If this is true, what is the minimum number of steps sufficient for the return?",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19837,
        "difficulty": null
      },
      {
        "id": "d437b159e27f157f8df6e4b6a491506b333e85f55f9f377cda7806f8e738b454",
        "domain": "numina",
        "question": "17*. From the formula for the circumference: $C=2 \\pi r$, we get: $\\pi=$ $=\\frac{C}{2 r}$ or $\\frac{1}{\\pi}=\\frac{2 r}{C}$. Consider a square with a perimeter of 2 and sequentially find the values of $h_{p}$ and $r_{p}$ using the formulas from the previous problem. By repeatedly performing the calculations, achieve that the values of $h_{p}$ and $r_{p}$ coincide in the first five decimal places. This will find the value of $\\frac{1}{\\pi}=\\frac{2 r}{\\bar{c}} \\approx \\frac{2 r}{2}=r$ with the corresponding accuracy. (This method of calculating $\\pi$ is called the method of isoperimeters.)",
        "answer": "0.3183098",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44266,
        "difficulty": null
      },
      {
        "id": "c6b2a45b94ffe0708a15e46f1202e7b9736a82fe9ff1eea79caf401cdfd35112",
        "domain": "numina",
        "question": "15. The famous Goldbach Conjecture can be stated as: Any even number greater than 2 can be expressed as the sum of two prime numbers. Express the even number 88 as the sum of two prime numbers, how many ways are there? $(a+b$ and $b+a$ are considered the same representation method)",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144132,
        "difficulty": null
      },
      {
        "id": "fdfa0cb158ace57e54fe61ba085cb2341aa3c3364f647d27cdea2740fa3d77d7",
        "domain": "numina",
        "question": "79. A small boat sails upstream from downstream port $A$, and at a point 4 kilometers away from port $A$, it encounters a log drifting with the current. The boat continues its journey to reach upstream port $B$ and immediately turns around, arriving at port $A$ at the same time as the log. Given that the speed of the current is 2 meters/second, and the speed of the boat in still water is 8 meters/second, the distance between ports $A$ and $B$ is $\\qquad$ kilometers.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108240,
        "difficulty": null
      },
      {
        "id": "840009f2863a831aae300e3aa5c1a5d4cf56b0be3d83b0546fb72aa6f80ef308",
        "domain": "numina",
        "question": "5. As shown in Figure 2, in $\\triangle A B C$, $\\angle A B C=120^{\\circ}$,\n$A B=B C=2$. Take a point $D$ on side\n$A C$ (excluding $A$ and $C$), and fold $\\triangle A B D$\nalong line segment $B D$, to get\n$\\triangle P B D$. When plane\n$P B D \\perp$ plane $A B C$, the maximum distance from point $P$ to plane $A B C$ is $\\qquad$",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142090,
        "difficulty": null
      },
      {
        "id": "4d35fee1d69115ba0d038be0d886da7d1b1e5f543d194afde2c8b9194a251772",
        "domain": "numina",
        "question": "1. The wolf saw a roe deer several meters away from him and chased after her along a straight forest path. The wolf's jump is $22\\%$ shorter than the roe deer's jump. Both animals jump at a constant speed. All the roe deer's jumps are of the same length, and the wolf's jumps are also equal to each other. There is a period of time during which both the wolf and the roe deer make a certain whole number of jumps. Each time, it turns out that the wolf has made $t\\%$ more jumps than the roe deer. Find the greatest integer value of $\\mathrm{t}$, for which the wolf will not be able to catch the roe deer.",
        "answer": "28",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 21776,
        "difficulty": null
      },
      {
        "id": "5969d3e81ffd499faacbf7cdd2401d938d97ee3a040497b2af8d32441a24c0f5",
        "domain": "numina",
        "question": "Bogdanov I.I.\n\nIn the language of the AU tribe, there are two letters - \"a\" and \"u\". Some sequences of these letters are words, and each word contains no fewer than one and no more than 13 letters. It is known that if you write down any two words in a row, the resulting sequence of letters will not be a word. Find the maximum possible number of words in such a language.",
        "answer": "16256",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57178,
        "difficulty": null
      },
      {
        "id": "42cc1e82211892524b5e2c6fd6da36761b908bd7fdd5d59c10ca05ea450c9a93",
        "domain": "numina",
        "question": "4. The random variable $a$ is uniformly distributed on the interval $[-2 ; 1]$. Find the probability that all roots of the quadratic equation $a x^{2}-x-4 a+1=0$ in absolute value exceed 1.",
        "answer": "\\frac{7}{9}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 2369,
        "difficulty": null
      },
      {
        "id": "00d55f1d9866ecf8927942392b5648d83acb9c8f7ad7768bc9fb74d0fce6def9",
        "domain": "numina",
        "question": "5-1. Solve the inequality\n\n$$\n\\sqrt{3 x-7}-\\sqrt{3 x^{2}-13 x+13} \\geqslant 3 x^{2}-16 x+20\n$$\n\nIn your answer, specify the sum of all integer values of $x$ that satisfy the inequality.",
        "answer": "3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 5440,
        "difficulty": null
      },
      {
        "id": "969d8b196089f2d3d6c30a64a63eb0706d29462445052fb04a1a81822f84984f",
        "domain": "numina",
        "question": "4. How many natural numbers are there that are less than or equal to the number 2019, and which are not divisible by any of the numbers 6, 9, and 15?",
        "answer": "1526",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86472,
        "difficulty": null
      },
      {
        "id": "dfbb3f3e11f8ef0622d3a1760cf00f94b7ed8637f8f52ecbddcea854a4ffcf05",
        "domain": "numina",
        "question": "3.1.3 * Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence, $a_{1}=19, a_{26}=-1$. Let $A=a_{n}+a_{n+1}+\\cdots+a_{n+6}$, where $n$ is a positive integer, find the minimum value of $|A|$.",
        "answer": "\\frac{7}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125741,
        "difficulty": null
      },
      {
        "id": "04f38194ad18e2ed75b73bc1f84e70830c1bc5ec0a6154ea2406e05521d2cc93",
        "domain": "numina",
        "question": "## Task 3 - 290913\n\nIn a counting-out game, 11 children stand in a circle. One of these children recites the counting-out rhyme; with each syllable, one child is counted in a clockwise direction. The player who recites the counting-out rhyme is also included in the counting. The counting-out rhyme has 15 syllables. The child on whom the last syllable falls leaves the circle; the counting resumes with the next child, starting again from the beginning of the counting-out rhyme. This counting and elimination continues until only one player remains in the circle; this player is the winner.\n\na) At which child should the counting player start if he wants to win himself? (To formulate the answer, number the children and assign, for example, the counting player the number 1.)\n\nb) If you have the opportunity to work on a (small) computer, you should write a program that solves task a) for counting-out games with $k$ children and a counting-out rhyme of $s$ syllables.",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 184728,
        "difficulty": null
      },
      {
        "id": "dc51df1748217a04eb8d61dddd399dc034e2052e16146386b6db2d98e7ec727b",
        "domain": "numina",
        "question": "7. The function $f(x)$ satisfies for all real numbers $x$:\n$$\nf(2-x)=f(2+x), f(5+x)=f(5-x) \\text {, and } f(0)=0\n$$\n\nThen $f(x)$ has at least $\\qquad$ zeros in the interval $[-21,21]$.",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145812,
        "difficulty": null
      },
      {
        "id": "a73fdd564361b688ba61e9a26c86387465fd059ac9dcae1a29bccd466699a197",
        "domain": "numina",
        "question": "75. The three-digit numbers $\\overline{a b c}$ and $\\overline{c b a}$ form a pair of reverse-order numbers (such as 123 and 321, 778 and 877). If the difference between the two numbers in a reverse-order pair is 297, how many such reverse-order pairs are there?",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 115871,
        "difficulty": null
      },
      {
        "id": "b6f9e3624cc4526c0f186db824985c5865b4cb0dc84da99235f14dd84a173bc6",
        "domain": "numina",
        "question": "4. The base of a right quadrilateral prism is a rhombus with side length 2 and an angle of $30^{\\circ}$, and the height is 1. A section is made through the base edge at an angle of $60^{\\circ}$ to the base. The area of this section is $\\qquad$ .",
        "answer": "\\frac{4\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 142850,
        "difficulty": null
      },
      {
        "id": "f3f3bb84a6b6e2155fd9e3519253e011f3106d8564d7a16810efa36b4c31516f",
        "domain": "numina",
        "question": "$9 \\cdot 4$ can be divided by 3, and the digits of each number are limited to $1,2,3$ (1,2,3 may not all be used) of all different natural numbers less than 200000, how many are there?\n(Shanghai, China High School Mathematics Competition, 1988)",
        "answer": "202",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 104872,
        "difficulty": null
      },
      {
        "id": "e87e9e85bdbacbb7668255347e31a688d1e286babe3e2702bec0a1c8616774fe",
        "domain": "numina",
        "question": "10,11\n\nThe edge of a regular tetrahedron is $a$. A plane $P$ passes through vertex $B$ and the midpoints of edges $A C$ and $A D$. A sphere is tangent to the lines $A B, A C, A D$ and that part of the plane $P$ which is inside the tetrahedron. Find the radius of the sphere. (Find all solutions).",
        "answer": "\\frac{\\sqrt{2}}{5\\\\sqrt{11}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60138,
        "difficulty": null
      },
      {
        "id": "4481309ccebf27cdd346a4c35c45d1fd5b715fb2b220653ab0ade3c79916a6ab",
        "domain": "numina",
        "question": "[ [Divisibility of numbers. General properties]\n\nAuthors: Binkov A.d,, Raskina I.v.\n\nSeveral whole heads of cheese were stored in the warehouse. At night, rats came and ate 10 heads, and all of them ate equally. Several rats got sick from overeating. The remaining seven rats the next night finished off the remaining cheese, but each rat was able to eat only half as much cheese as the night before. How much cheese was originally in the warehouse?\n\n#",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68949,
        "difficulty": null
      },
      {
        "id": "1de37eda44ab728a90937643fe707b93a9adff86c75290ea7bf62c384c3cb1bb",
        "domain": "numina",
        "question": "78. The Hope Primary School is 48 km away from the county town, part of which is an uphill road, and the rest is a downhill road. Principal Zhang rides a bicycle from the school to the county town, taking 4.2 hours to go and 3.8 hours to return. It is known that Principal Zhang rides a bicycle uphill at a speed of 10 km per hour, so he rides a bicycle downhill at a speed of $\\qquad$ km per hour.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125126,
        "difficulty": null
      },
      {
        "id": "592711a085f84b15f7195f9f27639176523fbf09de74797b83d59da8f65d87a7",
        "domain": "numina",
        "question": "8. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then $f\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)=$",
        "answer": "500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96798,
        "difficulty": null
      },
      {
        "id": "22d1f128cfd5553bd6334dc7748cb83025f34c1afa2ebaf9c32dfdd3278e488c",
        "domain": "numina",
        "question": "463. From two ports A and B, two steamships set off simultaneously towards each other across the sea. The speed of each is constant. The first steamship arrived at B 16 hours after the meeting, while the second arrived at A 25 hours after the meeting. How long does it take for each steamship to travel the entire distance?",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 34697,
        "difficulty": null
      },
      {
        "id": "6d81e81d975eb0488838ca43696be855a9de10d98922548d0085bd2450fb9b11",
        "domain": "numina",
        "question": "At the mountain cabin, our coach said: \"If we continue at this comfortable pace of $4 \\mathrm{~km}$ per hour, we will arrive at the station 45 minutes after our train has left.\"\n\nThen he pointed to a group that was just passing us: \"They are using poles, and thus achieve an average speed of $6 \\mathrm{~km}$ per hour. They will be at the station half an hour before our train leaves.\"\n\nHow far was the station from the mountain cabin?\n\n(M. Volfová)\n\nHint. Recall the relationships between average speed, total distance, and required time.",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76365,
        "difficulty": null
      },
      {
        "id": "b8b8a490461ad6d59fbdf057426979b83ead7d8c838b081dff748891758d0aa2",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{1} x^{2} \\sqrt{1-x^{2}} d x\n$$",
        "answer": "\\frac{\\pi}{16}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46804,
        "difficulty": null
      },
      {
        "id": "bab3f0e83b68f8fed78c656e125f53fa147f78aab5df93a9b511232b72c6f3dc",
        "domain": "numina",
        "question": "34. A company gathered for a meeting. Let's call a person sociable if in this company they have at least 20 acquaintances, and at least two of them are acquainted with each other. Let's call a person shy if in this company they have at least 20 strangers, and at least two of them are strangers to each other. It turned out that in the gathered company there are neither sociable nor shy people. What is the maximum number of people that can be in it\n\n$\\left(\\right.$ Folklore $^{(2)}$",
        "answer": "40",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18945,
        "difficulty": null
      },
      {
        "id": "015725b2c782405515fccd186f6456df6ff33a8f8f29081feed3ac808821aac6",
        "domain": "numina",
        "question": "7.6. Find the volume of a regular triangular pyramid, the midpoint of the height of which is at a distance of 2 and $\\sqrt{5}$ from the lateral face and the lateral edge, respectively. Round your answer to two decimal places if necessary.",
        "answer": "485.42",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6425,
        "difficulty": null
      },
      {
        "id": "ae3f0e6439c0c8d9ab8fbcf5ee2c1c0db6846a507b49a9ca98645f14ef99d87f",
        "domain": "numina",
        "question": "13. Let $n$ be the smallest positive integer of 4 digits greater than or equal to 2016 that has the following property: there exists a positive integer $S$ such that\n\n$$\nS=\\sqrt{a+\\sqrt{b+\\sqrt{c+\\sqrt{d+S}}}}\n$$\n\nwhere $a, b, c, d$ are, in order, the thousands, hundreds, tens, and units digits of $n$. What is the value of $n$?",
        "answer": "2167",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181468,
        "difficulty": null
      },
      {
        "id": "5f13c06ac5fa111c565a7ca78d34e0e0773d02c0102e1e4adeba742ccd5226a2",
        "domain": "numina",
        "question": "11. Given real numbers $x_{1}, x_{2}, x_{3}$ satisfy $x_{1}^{2}+x_{2}^{2}+x_{3}^{2}+x_{1} x_{2}+x_{2} x_{3}=2$. Then the maximum value of $\\left|x_{2}\\right|$ is $\\qquad$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114675,
        "difficulty": null
      },
      {
        "id": "1606c37da20f785bd35c296c2969ec3680b186119443c97be4f80a906aeebd11",
        "domain": "numina",
        "question": "2. In an airplane with 62 rows, each having 6 seats, passengers have seated themselves in such a way that in no 2 rows are seats occupied in the same positions. What is the maximum number of passengers that can be in the airplane?",
        "answer": "191",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 162759,
        "difficulty": null
      },
      {
        "id": "04c7fbda038fe55e3038b25feb72189e9ede289133c8089485b4a1d73b6343ed",
        "domain": "numina",
        "question": "Task 2. On an island, there live knights who always tell the truth, and liars who always lie. One day, 30 inhabitants of this island sat around a round table. Each of them said one of two phrases: \"My left neighbor is a liar\" or \"My right neighbor is a liar.\" What is the smallest number of knights that can be at the table?",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 4427,
        "difficulty": null
      },
      {
        "id": "28c204001ad38da948d4c3b2c845d7da4fa1b543243b61378885b76d6b7fdcac",
        "domain": "numina",
        "question": "Touching Circles\n\nTwo circles touch each other externally at point $A$. Through point $B$ on their common tangent $A B$, two lines are drawn, one intersecting the first circle at points $M$ and $N$, and the other intersecting the second circle at points $P$ and $Q$. It is known that $A B=6, B M=9, B P=5$. Find the ratio of the areas of triangles $M N O$ and $P Q O$, where point $O$ is the intersection of lines $M P$ and $N Q$.",
        "answer": "\\frac{625}{121}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66100,
        "difficulty": null
      },
      {
        "id": "ac27dc7e32afb470c9e0b76a30e3527ec5e3b0e589cfe5e4dd68fe144e6a05b2",
        "domain": "numina",
        "question": "10.055. A circle is inscribed in an isosceles triangle with a vertex angle of $120^{\\circ}$ and a lateral side of $a$ (Fig. 10.55). Find the radius of this circle.",
        "answer": "\\frac{\\sqrt{3}(2-\\sqrt{3})}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51176,
        "difficulty": null
      },
      {
        "id": "419373142649b32f015997bf88d05de8f34f806bf907f4a1adc4988b0488624c",
        "domain": "numina",
        "question": "7.5. Vasya had 101 real coins. One of them was replaced with a fake one (it differs in weight, but it is unknown whether it is lighter or heavier than a real coin). Vasya wants to find as many real coins as possible in one weighing on a balance scale without weights. What should he do and how many real coins will he be able to find?",
        "answer": "50",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12066,
        "difficulty": null
      },
      {
        "id": "e5a4441d645a612601ae98311d00ae60d648c93b2cebda75ef9c0492a93d3651",
        "domain": "numina",
        "question": "4. Friends Petya, Vasya, and Kolya live in the same entrance. Climbing from the 1st to the 3rd floor, Petya passes a number of steps that is a multiple of four. Vasya, climbing from the 1st to the 6th floor, passes a number of steps that is a multiple of nine. How many steps will Kolya pass, who climbs from the 1st to the 9th floor, if the number of steps between floors is the same and less than $22?$",
        "answer": "144",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17103,
        "difficulty": null
      },
      {
        "id": "e91e4649c6a75f26359f4c934727e488fb9144b013d72764913cdaa367705bb0",
        "domain": "numina",
        "question": "H6. The diagram shows four identical white triangles symmetrically placed on a grey square. Each triangle is isosceles and right-angled.\nThe total area of the visible grey regions is equal to the total area of the white triangles.\nWhat is the smallest angle in each of the (identical) grey triangles in the diagram?",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 155232,
        "difficulty": null
      },
      {
        "id": "62dff81fbd83467bba6d0e3a730a4f96dadba0fcee266be103290d6b12d1e8b2",
        "domain": "numina",
        "question": "10.2.1. (12 points) Find the greatest integer value of $a$ for which the equation\n\n$$\n\\sqrt[3]{x^{2}-(a+7) x+7 a}+\\sqrt[3]{3}=0\n$$\n\nhas at least one integer root.",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6540,
        "difficulty": null
      },
      {
        "id": "629e01e101eca62ccb71d0e24d0644ea07044544a7b301623c49aeccfa307f16",
        "domain": "numina",
        "question": "3. In a cube $A B C D-$ $A_{1} B_{1} C_{1} D_{1}$ with edge length 2, $M$ and $N$ are the midpoints of edges $B B_{1}$ and $B_{1} C_{1}$, respectively. If $P$ is a moving point in the plane $D M N$, and the distance from point $P$ to the plane $B C C_{1} B_{1}$ is equal to the length of $P D$, the eccentricity of the trajectory of point $P$ is $\\qquad$.",
        "answer": "\\frac{2\\sqrt{34}}{17}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 121506,
        "difficulty": null
      },
      {
        "id": "82ef084242301e67dd5037a656107909cfddbf8a094d1eba2b1d7aba908475bf",
        "domain": "numina",
        "question": "7. The volume of a cube in cubic metres and its surface area in square metres is numerically equal to four-thirds of the sum of the lengths of its edges in metres.\nWhat is the total volume in cubic metres of twenty-seven such cubes?",
        "answer": "216",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156475,
        "difficulty": null
      },
      {
        "id": "34286395264fde4e8eb2da63572d6ecdeb63091dc0bb572e4e999bd75c85dc36",
        "domain": "numina",
        "question": "## Task 4 - 330834\n\nOn a route $A B$, a cyclist $X$ travels from $A$ to $B$, and a second cyclist $Y$ travels from $B$ to $A$. Both start at the same time. Upon reaching $B$ or $A$, they immediately turn around and travel the same route back to $A$ or $B$, respectively, and then end their journey. It is assumed that each cyclist maintains a constant speed and that the time used for turning around can be neglected.\n\nOn the outbound trip, they met 30 minutes after the start at a point that is $7.5 \\mathrm{~km}$ from $A$. Another 30 minutes later, the cyclists were together again at a point between $A$ and $B$.\n\nDetermine all possibilities for how, according to this description,\n\na) the length of the route $A B$,\n\nb) the speeds of the cyclists $X$ and $Y$ can be!",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184312,
        "difficulty": null
      },
      {
        "id": "b8c2110765a2d100f3a0c84f15744c9864613b51a5e9ccd3394afb09d65ea6be",
        "domain": "numina",
        "question": "In the plane, consider the set $S$ of points with coordinates $(x, y)$, where $x, y \\in$ $\\{1,2, \\ldots, 2013\\}$. Two points in $S$ are called neighbors if they are at a distance of 1 from each other. Every second, a fly and some spiders move on the points of $S$ as follows: first, the fly either stays still or moves to a neighboring point of the one it is on. Then, each spider either stays still or moves to a neighboring point of the one it is on. Several spiders can be on the same point at the same time, and the fly as well as the spiders know each other's positions.\na) Determine the smallest integer $k$ for which $k$ spiders can always catch the fly in a finite time, regardless of the initial positions of the fly and the spiders on the points of $S$.\nb) Answer the same question if we assume this time that we are in the usual space and that $S$ is the set of points with coordinates $(x, y, z)$ where $x, y, z \\in\\{1,2, \\ldots, 2013\\}$.",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 195555,
        "difficulty": null
      },
      {
        "id": "b333b56529f59181b410b7ea98d9a63841bad98460dd5161e8bbc05d082e9634",
        "domain": "numina",
        "question": "1. (Option 1) The decimal representation of the natural number $n$ contains sixty-three digits. Among these digits, there are twos, threes, and fours. No other digits are present. The number of twos is 22 more than the number of fours. Find the remainder when $n$ is divided by 9.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9199,
        "difficulty": null
      },
      {
        "id": "0a4e8a6a6d4a04a6b9e5910c35a8e74e513151288bcc3c81b5f0263c431c6b82",
        "domain": "numina",
        "question": "Konyagin S.V.\n\nGiven a square with side length 1. From it, four corners are cut off - four triangles, each of which has two sides along the sides of the square and these sides are $1 / 3$ of their lengths. With the resulting 8-sided polygon, the same operation is performed: from each vertex, a triangle is cut off, two sides of which are $1 / 3$ of the corresponding sides of the 8-sided polygon, and so on. A sequence of polygons is obtained (each contained in the previous one). Find the area of the figure that is the intersection of all these polygons (i.e., formed by the points belonging to all polygons).",
        "answer": "\\frac{5}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67961,
        "difficulty": null
      },
      {
        "id": "0fc1c11ce273f653831a5b0766fe361153e406a37a66bc8950d91ad9988bf08d",
        "domain": "numina",
        "question": "9. (40 points) For real numbers $a, b$ and $c$ it is known that $a b + b c + c a = 3$. What values can the expression $\\frac{a\\left(b^{2}+3\\right)}{a+b}+\\frac{b\\left(c^{2}+3\\right)}{b+c}+\\frac{c\\left(a^{2}+3\\right)}{c+a}$ take?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8341,
        "difficulty": null
      },
      {
        "id": "b9c5c867f51e8e4927123a38a1e8cbaf800e7ab279da616328dc4029cf4ceeb8",
        "domain": "numina",
        "question": "6. Let $S$ be the set of all rational numbers in $\\left(0, \\frac{5}{8}\\right)$, and for a simplified fraction $\\frac{q}{p} \\in S, (p, q)=1$, define the function $f(q p)=\\frac{q+1}{p}$. Then the number of roots of $f(x)=\\frac{2}{3}$ in $S$ is $\\qquad$.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 110491,
        "difficulty": null
      },
      {
        "id": "9fb3c7924b7c8610d6923b72540b7b19646e8f17ab395325eae8adc12f7ec0c8",
        "domain": "numina",
        "question": "19. The mammal with the longest lifespan on Earth is the bowhead whale. There are 10 bowhead whales, and this year their ages are consecutive natural numbers starting from 209 to 218. Then, when the youngest bowhead whale is $\\qquad$ years old, the sum of the ages of these 10 bowhead whales will be 2020 years less than the sum of their ages this year.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119768,
        "difficulty": null
      },
      {
        "id": "27f6921b6d2c38f70a0a29f009e8f534d0c90068ca2b9fa5b8b07cf0a9720824",
        "domain": "numina",
        "question": "[ Doubling the Median ] [ Law of Cosines ]\n\nDetermine the angle $A$ between the sides of lengths 2 and 4, if the median drawn from vertex $A$ is $\\sqrt{7}$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62544,
        "difficulty": null
      },
      {
        "id": "3d391101b8b18dd5b2a0ee3a81a47c65ab1311126012fb17963883a2e81f0ad2",
        "domain": "numina",
        "question": "5. Let two regions on the Cartesian plane be $M=\\left\\{(x, y) \\in \\mathbf{R}^{2} \\mid 0 \\leqslant y \\leqslant \\min (2 x, 3-x)\\right\\}, N=$ $\\left\\{(x, y) \\in \\mathbf{R}^{2} \\mid t \\leqslant x \\leqslant t+2\\right\\}$, and let the area of the intersection of $M$ and $N$ be $f(t)$. When $0 \\leqslant t \\leqslant 1$, the expression for $f(t)$ is $\\qquad$.",
        "answer": "-\\frac{3}{2}^{2}++\\frac{5}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 144733,
        "difficulty": null
      },
      {
        "id": "1201bf7f74dcc9098a1691adc3646a1176fa1fe1501ca258441a4428605a4a4b",
        "domain": "numina",
        "question": "3. Solve the equation $2^{1-2|x|}+2 \\cdot 4^{1+|x|}=17$.",
        "answer": "\\\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10352,
        "difficulty": null
      },
      {
        "id": "e407f6490dd261c4783bc0afb7e300e7feb4c2e53ed333f0d6a95e243e141b7a",
        "domain": "numina",
        "question": "240. Given a cube $A B C D A_{1} B_{1} C_{1} D_{1}$ with edge $a$. Find the radius of the smallest sphere that touches the lines $A B_{1}, B_{1} C$, $C D$ and $D A$.",
        "answer": "\\sqrt{1-\\frac{\\sqrt{2}}{2}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29942,
        "difficulty": null
      },
      {
        "id": "25221c264eed2f56fbe45472a8df52b40a8b77062fc5bd98abc32ce78feb453a",
        "domain": "numina",
        "question": "57 Let $f(x)=2^{x}, g(x)=\\log _{\\sqrt{2}} 8 x$, then the value of $x$ that satisfies $f[g(x)]=g[f(x)]$ is\n\nLet's translate the problem and solution step by step.\n\n### Problem:\nLet \\( f(x) = 2^x \\) and \\( g(x) = \\log_{\\sqrt{2}} (8x) \\). Find the value of \\( x \\) that satisfies \\( f[g(x)] = g[f(x)] \\).\n\n### Solution:\n1. **Express \\( f[g(x)] \\):**\n   \\[\n   f[g(x)] = f\\left( \\log_{\\sqrt{2}} (8x) \\right)\n   \\]\n   Since \\( f(x) = 2^x \\), we have:\n   \\[\n   f\\left( \\log_{\\sqrt{2}} (8x) \\right) = 2^{\\log_{\\sqrt{2}} (8x)}\n   \\]\n\n2. **Simplify \\( 2^{\\log_{\\sqrt{2}} (8x)} \\):**\n   Recall that \\( \\log_{\\sqrt{2}} (8x) = \\frac{\\log_2 (8x)}{\\log_2 (\\sqrt{2})} \\). Since \\( \\log_2 (\\sqrt{2}) = \\frac{1}{2} \\), we get:\n   \\[\n   \\log_{\\sqrt{2}} (8x) = 2 \\log_2 (8x)\n   \\]\n   Therefore:\n   \\[\n   2^{\\log_{\\sqrt{2}} (8x)} = 2^{2 \\log_2 (8x)} = (2^{\\log_2 (8x)})^2 = (8x)^2 = 64x^2\n   \\]\n\n3. **Express \\( g[f(x)] \\):**\n   \\[\n   g[f(x)] = g(2^x)\n   \\]\n   Since \\( g(x) = \\log_{\\sqrt{2}} (8x) \\), we have:\n   \\[\n   g(2^x) = \\log_{\\sqrt{2}} (8 \\cdot 2^x) = \\log_{\\sqrt{2}} (8 \\cdot 2^x)\n   \\]\n\n4. **Simplify \\( \\log_{\\sqrt{2}} (8 \\cdot 2^x) \\):**\n   \\[\n   \\log_{\\sqrt{2}} (8 \\cdot 2^x) = \\log_{\\sqrt{2}} (2^3 \\cdot 2^x) = \\log_{\\sqrt{2}} (2^{3+x})\n   \\]\n   Using the change of base formula:\n   \\[\n   \\log_{\\sqrt{2}} (2^{3+x}) = \\frac{\\log_2 (2^{3+x})}{\\log_2 (\\sqrt{2})} = \\frac{3 + x}{\\frac{1}{2}} = 2(3 + x) = 6 + 2x\n   \\]\n\n5. **Set \\( f[g(x)] \\) equal to \\( g[f(x)] \\):**\n   \\[\n   64x^2 = 6 + 2x\n   \\]\n\n6. **Solve the equation \\( 64x^2 = 6 + 2x \\):**\n   \\[\n   64x^2 - 2x - 6 = 0\n   \\]\n   This is a quadratic equation. We can solve it using the quadratic formula \\( x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\), where \\( a = 64 \\), \\( b = -2 \\), and \\( c = -6 \\):\n   \\[\n   x = \\frac{-(-2) \\pm \\sqrt{(-2)^2 - 4 \\cdot 64 \\cdot (-6)}}{2 \\cdot 64} = \\frac{2 \\pm \\sqrt{4 + 1536}}{128} = \\frac{2 \\pm \\sqrt{1540}}{128} = \\frac{2 \\pm 2\\sqrt{385}}{128} = \\frac{1 \\pm \\sqrt{385}}{64}\n   \\]\n\n7. **Check the solutions:**\n   \\[\n   x = \\frac{1 + \\sqrt{385}}{64} \\quad \\text{and} \\quad x = \\frac{1 - \\sqrt{385}}{64}\n   \\]\n   Since \\( x \\) must be positive, we discard \\( x = \\frac{1 - \\sqrt{385}}{64} \\).\n\nThus, the value of \\( x \\) that satisfies the equation is:\n\\[\n\\boxed{\\frac{1 + \\sqrt{385}}{64}}\n\\]",
        "answer": "\\frac{1+\\sqrt{385}}{64}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136045,
        "difficulty": null
      },
      {
        "id": "33a6ff0f895b5e6d9611a3d2be669c60b227e2868dd871841cafb4f5d8913dd2",
        "domain": "numina",
        "question": ". For what value of $k$ do we have the identity (true for all $a, b, c$):\n\n$$\n(a+b)(b+c)(c+a)=(a+b+c)(a b+b c+c a)+k a b c ?\n$$",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 82685,
        "difficulty": null
      },
      {
        "id": "7192ace3a58c1f615b5531caa8083101d58f0ddf07c090d46c380130072038d2",
        "domain": "numina",
        "question": "Bakayev E.V.\n\nA secret base is surrounded by a transparent winding fence in the shape of a non-convex polygon, with a swamp outside. A straight power line consisting of 36 poles runs through the swamp, some of which are outside the base, and some are inside. (The power line does not pass through the vertices of the fence.) A spy walks around the base along the fence from the outside, keeping the fence always to his right. Each time he reaches the power line, he counts how many poles are to his left (he can see all of them). By the time the spy has walked around the entire fence, he has counted a total of 2015 poles. How many poles are inside the base?",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62508,
        "difficulty": null
      },
      {
        "id": "0ed327eea676b468d60e04f879e7f9dee358c538064bf5efe3264a37051fd776",
        "domain": "numina",
        "question": "G5.4 Let $S_{1}, S_{2}, \\ldots, S_{10}$ be the first ten terms of an A.P., which consists of positive integers.\nIf $\\mathrm{S}_{1}+\\mathrm{S}_{2}+\\ldots+\\mathrm{S}_{10}=55$ and $\\left(\\mathrm{S}_{10}-\\mathrm{S}_{8}\\right)+\\left(\\mathrm{S}_{9}-\\mathrm{S}_{7}\\right)+\\ldots+\\left(\\mathrm{S}_{3}-\\mathrm{S}_{1}\\right)=d$, find $d$.",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152811,
        "difficulty": null
      },
      {
        "id": "a4d52b2c2bb10b0a59abf05e7b8783ef570708458761a2b7973a729b0bfae86a",
        "domain": "numina",
        "question": "## Task 2.\n\nLet $a \\geqslant 2018$ be a real number. In each of 2018 jars, there is a finite number of balls, with the mass of each ball being of the form $a^{k}$, where $k \\in \\mathbb{Z}$. The total mass of the balls in each jar is the same. What is the minimum number of balls of the same mass among the balls in the jars?",
        "answer": "2018",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 89723,
        "difficulty": null
      },
      {
        "id": "dcaf1c2599dbe52e84a0b447c6cf4e5231e5db587a64857b17545007ecc37a75",
        "domain": "numina",
        "question": "G3.2 In Figure $3, A$ and $B$ are two cones inside a cylindrical tube with length of 20 and diameter of 6 . If the volumes of $A$ and $B$ are in the ratio $3: 1$ and $b$ is the height of the cone $\\mathrm{B}$, find the value of $b$.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152917,
        "difficulty": null
      },
      {
        "id": "55a02345e2c6c502d61a2ff31b3e281a33d690c871ea634062d693360198125a",
        "domain": "numina",
        "question": "14. A number $N$ is the product of three distinct primes. How many distinct factors does $N^{5}$ have?",
        "answer": "216",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156484,
        "difficulty": null
      },
      {
        "id": "bc261cb86ce899c7d774655563581069b842eeabdd0132910814d132835c010e",
        "domain": "numina",
        "question": "## Task 1 - 050911\n\nA turner needs half an hour to manufacture a certain part. Since several identical parts need to be produced, he is considering building a fixture that would allow each such part to be manufactured in 20 minutes. The construction of this fixture would take 4 hours.\n\nHow many parts at least need to be produced for the construction of the fixture to result in time savings?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184416,
        "difficulty": null
      },
      {
        "id": "e33503117a316af6d5698b13e6ae3afa44acb6457995f86e2c756482416eab4a",
        "domain": "numina",
        "question": "5. By arranging sticks in a hexagonal grid, students create shapes called hexes. In the image, a hex is shown that has five hexagons in the center (highlighted in the image). Sticks are sold in boxes containing 45 sticks each, and the price of one box is 1 euro and 25 cents.\n\nWhat is the minimum amount that students must spend to buy enough sticks to form a hex that has 13 hexagons in the center?\n\n![](https://cdn.mathpix.com/cropped/2024_05_30_e5ca9b3e8971fb159cf9g-5.jpg?height=483&width=448&top_left_y=110&top_left_x=1455)",
        "answer": "12.50",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 90009,
        "difficulty": null
      },
      {
        "id": "57c8f62df585a8df3f0d8fa2324f45845618d0b84c335b6e2618e605466712c8",
        "domain": "numina",
        "question": "[ [rectangular parallelepipeds]\n\nThe diagonals of the faces of a rectangular parallelepiped are $\\sqrt{3}, \\sqrt{5}$, and 2. Find its volume.",
        "answer": "\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63562,
        "difficulty": null
      },
      {
        "id": "398f58de017c07979398ba7a12b8cdbfebe7df4e2e8539b19bffc890071373e0",
        "domain": "numina",
        "question": "4. Let $n>1$ be a natural number. An equilateral triangle with side length $n$ is divided by lines parallel to its sides into congruent equilateral smaller triangles with side length 1. The number of smaller triangles that have at least one side on the side of the original triangle is 1 less than the number of all other smaller triangles. Determine all such $n$.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86940,
        "difficulty": null
      },
      {
        "id": "5f7473e4e5dee944ea22d4db8d25eb56f47f60ccd68998767e46700288b81ff3",
        "domain": "numina",
        "question": "12. A complicated choreography involves a row of 7 dancers equally spaced on a flat stage; in front of the row of dancers, there is an identical, parallel row of the same number of male dancers. The choreographer wants to assign to each dancer some male dancers in such a way that, by drawing on the stage the segment connecting the position of each dancer to that of the male dancers assigned to her, there are no segments that intersect except at the ends. She also wants the total number of segments drawn to be 13. How many ways are there to make the assignments that satisfy these properties?\n(A) 128\n(B) 343\n(C) 486\n(D) 924\n(E) 1716\n\n## Numerical answer problems -5 points",
        "answer": "924",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 181481,
        "difficulty": null
      },
      {
        "id": "e9aa7af751a56de50313a9587e6110b4307e670df872ff02fdb05967cec58a72",
        "domain": "numina",
        "question": "5. As shown in the figure, $\\angle B=30^{\\circ}, \\angle A=60^{\\circ}, \\angle D=20^{\\circ}$, then the degree of $\\angle B C D$ (the angle with the arc part in the figure) is\n$\\qquad$ $\\circ$ .",
        "answer": "250",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 140209,
        "difficulty": null
      },
      {
        "id": "ddc32ffcba85829527b9a759b1736359defb6464cff0e1f13fb8a7d677c4926a",
        "domain": "numina",
        "question": "At the end of the quarter, Vovochka wrote down his current grades for singing in a row and placed a multiplication sign between some of them. The product of the resulting numbers turned out to be 2007. What grade does Vovochka get for singing in the quarter? (\"Kol\" is not given by the singing teacher.)\n\n#",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 56633,
        "difficulty": null
      },
      {
        "id": "e5830dc00aa538ca5b6a924f75836dc6d53f12badb8fae00dd40a820776162a2",
        "domain": "numina",
        "question": "9. Let the real number $a<-1$, and the variable $x$ satisfies $x^{2}+a x \\leqslant-x$, and the minimum value of $x^{2}+a x$ is $-\\frac{1}{2}$. Then $a=$ $\\qquad$ .",
        "answer": "-\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113241,
        "difficulty": null
      },
      {
        "id": "7230ea6e196a83bb55271c76e9615fdc3dae0047f8b0a9cce71d1ec8599455d1",
        "domain": "numina",
        "question": "6. On domino tiles, there are pairs of numbers from 0 to 6 (including numbers 0 and 6) located on two fields. All numbers, except zero, are represented by the corresponding number of dots. An empty field without dots represents zero. In a complete set of domino tiles, there are also tiles with the same values on both fields (for example: 1 - 1, ...). How many dots are there in total in the entire set of domino tiles?",
        "answer": "168",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86775,
        "difficulty": null
      },
      {
        "id": "a3eb7c25d02a00d28a007b493d5c51112d67d5929950785e59bbca2296e567e5",
        "domain": "numina",
        "question": "On Mr. Chernick's farm, they had three hens. The first laid one egg every day, the second laid one egg every other day, and the third laid one egg every third day. Mr. Chernick bought two new hens at the market, which also laid eggs completely regularly - always one after a few days, with one of them laying twice as often as the other.\n\nMrs. Chernick calculated that all five hens laid a total of 155 eggs in 60 days. How often did the new hens lay eggs?\n\n(L. Hozová)",
        "answer": "42",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76304,
        "difficulty": null
      },
      {
        "id": "14983f9814f354c49e331e072d2038a5672acf092eb5d8b475e337b18d1b4085",
        "domain": "numina",
        "question": "13th Australian 1992 Problem A1 P, Q, R are adjacent vertices of a regular 9-gon with center O. M is the midpoint of the segment joining O to the midpoint of QR and N is the midpoint of PQ. Find the ∠ONM.",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150643,
        "difficulty": null
      },
      {
        "id": "64b8888acf58794f692c1ca2fbbe332b87441a46111b1def100183212b193aa0",
        "domain": "numina",
        "question": "14. Suppose there are 3 distinct green balls, 4 distinct red balls, and 5 distinct blue balls in an urn. The balls are to be grouped into pairs such that the balls in any pair have different colors. How many sets of six pairs can be formed?",
        "answer": "1440",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154615,
        "difficulty": null
      },
      {
        "id": "716eab06b56f28e0e384e47c2d65ea42726308968b3d3f91c0f669a965d2adbc",
        "domain": "numina",
        "question": "## 33. Reservoir\n\nThe reservoir has the shape of a rectangular parallelepiped, the width of which is half the length. It is filled to three eighths of its height. When 76 hl [1 hl (hectoliter) = 0.1 m³] is added, the water level rises by 0.38 m. After this, two sevenths of the reservoir remain to be filled.\n\nWhat is the height of the reservoir?",
        "answer": "1.12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42212,
        "difficulty": null
      },
      {
        "id": "5df34cf31bd03ad296c3ef50233a835463486ede579e41498e5567605ca30711",
        "domain": "numina",
        "question": "A triangle's three side lines are tangent to four circles whose radii are four consecutive elements of a geometric sequence. What is the largest angle of the triangle?",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166964,
        "difficulty": null
      },
      {
        "id": "5f05f6a43ad2ea3b084fb819fc780c99ae3084676ea57f28ba66f4354f932291",
        "domain": "numina",
        "question": "## Task 4\n\nAdd the numbers 1 to 10.",
        "answer": "55",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186558,
        "difficulty": null
      },
      {
        "id": "a114654472f9d55710d2f83cb0abbb857976060e98c22d153c42515e0618dadf",
        "domain": "numina",
        "question": "1. Let $x, y \\in \\mathbf{R}$, and the sets are\n$$\n\\begin{array}{l}\nA=\\left\\{(x, y) \\left\\lvert\\, x^{2}-3 x y+4 y^{2}=\\frac{7}{2}\\right.\\right\\}, \\\\\nB=\\{(x, y) \\mid k x+y=2\\}(k>0) .\n\\end{array}\n$$\n\nWhen $A \\cap B$ has only one element, $k=$ $\\qquad$",
        "answer": "\\frac{1}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116716,
        "difficulty": null
      },
      {
        "id": "3be20025c785f660363adfc379d32bc108223bec7be6cd52eb2a3bb98d265119",
        "domain": "numina",
        "question": "34th Eötvös 1930 Problem 2 A straight line is drawn on an 8 x 8 chessboard. What is the largest possible number of the unit squares with interior points on the line?",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150147,
        "difficulty": null
      },
      {
        "id": "8bd33f85bd2a09435f04e60b7e7ee74fce5215ee08c0ec5bfe03c6d7f3993e16",
        "domain": "numina",
        "question": "4. If the positive integer $a$ makes the maximum value of the function $f(x)=x+\\sqrt{13-2 a x}$ also a positive integer, then this maximum value is equal to . $\\qquad$",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133977,
        "difficulty": null
      },
      {
        "id": "e86b7da6eb2ba66d353e567e893fff1230cc9bde0104742a40f4d08072c6f3c5",
        "domain": "numina",
        "question": "4. The function $y=f(x)$ is defined on the set $(0,+\\infty)$ and takes positive values on this set. It is known that for any points $A$ and $B$ on the graph of the function, the areas of the triangle $A O B$ and the trapezoid $A B H_{B} H_{A}$ are equal to each other $\\left(H_{A}, H_{B} - \\text{the bases of the perpendiculars dropped from points } A \\text{ and } B \\text{ to the x-axis}\\right)$;\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_12c6f2bb73fb9dd0ba03g-1.jpg?height=380&width=469&top_left_y=2317&top_left_x=1453)\n\n## Interregional Olympiad for Schoolchildren Based on Departmental Educational Organizations in Mathematics\n\n$O$ - the origin). Find all such functions. Justify your solution. Given that $f(1)=4$, write the number $f(2)$ in the answer.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8755,
        "difficulty": null
      },
      {
        "id": "a6064e27a776826802596ec7b918b03954f55e9651b9dd1cb8bc0ab0e3511cca",
        "domain": "numina",
        "question": "4. According to the 2002 Government Work Report at the Fifth Session of the Ninth National People's Congress on March 5, 2002: \"In 2001, the Gross Domestic Product (GDP) reached 959.33 billion yuan, an increase of $7.3 \\%$ over the previous year.\" If the GDP grows at this rate every year during the Tenth Five-Year Plan period (2001-2005), then by the end of the Tenth Five-Year Plan, China's annual GDP will be approximately $\\qquad$ .",
        "answer": "127165",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96430,
        "difficulty": null
      },
      {
        "id": "3fb20398bfb0840789ad8912da79c1d576d1db999e9bef2955a0db77c3d97640",
        "domain": "numina",
        "question": "A circular plate with a weight of $P=1 \\mathrm{~kg}$ is supported at three points along its circumference, which form an isosceles triangle. The vertex angle of this triangle is $\\alpha=30^{\\circ}$. What load rests on each support point?",
        "answer": "0.46411",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 173514,
        "difficulty": null
      },
      {
        "id": "5b7c79f6c548f558c1611bdb3e35866ad54b22ae1c99e34a7e1c464b9df86091",
        "domain": "numina",
        "question": "## Task 4 - 220834\n\nA helicopter took off at 4:30 AM from a city $A$ and flew with a speed of $250 \\frac{\\mathrm{km}}{\\mathrm{h}}$ to a city $B$. It stayed there for 30 minutes and then flew back to $A$ on the same route with a speed of $200 \\frac{\\mathrm{km}}{\\mathrm{h}}$, arriving at $A$ on the same day at 11:45 AM.\n\nDetermine the length of the route from $A$ to $B$!",
        "answer": "750",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184184,
        "difficulty": null
      },
      {
        "id": "4ba3334fa9ff1f7e590aefc749df7febf92f3e4e4544195bb66300852e7cce3a",
        "domain": "numina",
        "question": "1. Let the sum of the digits of the natural number $x$ be $S(x)$. Then the solution set of the equation $x+S(x)+S(S(x))+S(S(S(x)))=2016$ is $\\qquad$ .",
        "answer": "1980",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105337,
        "difficulty": null
      },
      {
        "id": "8edb6e683ebe165f660d153f34d6ef524bd0343c3008d4b34696204d2f7815f7",
        "domain": "numina",
        "question": "$10 \\cdot 98$ Find the maximum positive integer $n$, such that there exist $n$ integer points in the coordinate plane, where any 3 points form a triangle, and the centroid of the triangle is not an integer point.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 141039,
        "difficulty": null
      },
      {
        "id": "a2c5ac1dd6340c2ec2e908dd346e5701ecdae90f4908e0066579f3d484048031",
        "domain": "numina",
        "question": "## Task A-4.1.\n\nFor $n \\in \\mathbb{N}$, we define the complex number\n\n$$\na_{n}=(1+i)\\left(1+\\frac{i}{\\sqrt{2}}\\right)\\left(1+\\frac{i}{\\sqrt{3}}\\right) \\cdots\\left(1+\\frac{i}{\\sqrt{n}}\\right)\n$$\n\nCalculate\n\n$$\n\\left|a_{1}-a_{2}\\right|+\\left|a_{2}-a_{3}\\right|+\\cdots+\\left|a_{2019}-a_{2020}\\right| \\text {. }\n$$",
        "answer": "2019",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89626,
        "difficulty": null
      },
      {
        "id": "6a690974dec3e3048027945948e1caad29d033834a1c7628759a05e4fad3c6aa",
        "domain": "numina",
        "question": "Birgit has a list of four numbers. Luciano adds these numbers together, three at a time, and gets the sums 415, 442, 396, and 325. What is the sum of Birgit's four numbers?",
        "answer": "526",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194049,
        "difficulty": null
      },
      {
        "id": "fff24676eee567cc509d0e8600108973f22a9892f13e9970c414dbdd6f574228",
        "domain": "numina",
        "question": "1. In the set $M=\\{1,2, \\cdots, 2018\\}$, the sum of elements whose last digit is 8 is $\\qquad$",
        "answer": "204626",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 114004,
        "difficulty": null
      },
      {
        "id": "f3a39f09a502da8bdb50c4fa84780108bd912e8991f4ff662342bee2835f4089",
        "domain": "numina",
        "question": "1. Given positive integers $a, b, c (a<b<c)$ form a geometric sequence, and\n$$\n\\log _{2016} a+\\log _{2016} b+\\log _{2016} c=3 \\text {. }\n$$\n\nThen the maximum value of $a+b+c$ is $\\qquad$ .",
        "answer": "4066273",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111969,
        "difficulty": null
      },
      {
        "id": "910a19f95b3958d67ac506a4e579e6dba1a688035230a33a775c14d0b0ca06c0",
        "domain": "numina",
        "question": "89. A white ball is placed into an urn containing two balls, after which one ball is randomly drawn from it. Find the probability that the drawn ball will be white, if all possible assumptions about the initial composition of the balls (by color) are equally likely.",
        "answer": "\\frac{2}{3}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32305,
        "difficulty": null
      },
      {
        "id": "7099a8b0a57f5372fdf4beded650ee70bdf108a6024eeb42942f86b5f8c48c95",
        "domain": "numina",
        "question": "5. In the sum $1+3+5+\\ldots+k$ of consecutive odd natural numbers, determine the largest addend $k$ such that $1+3+5+\\ldots+k=40000$.\n\n## Tasks worth 10 points:",
        "answer": "399",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86678,
        "difficulty": null
      },
      {
        "id": "a083f6457bab1eedd8442e70ba0aa534308e5a2b73f336230ef5fc2ab048a9dc",
        "domain": "numina",
        "question": "* Given the quadratic equation in $x$: $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$ has real roots, find the value of the real number $a$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104193,
        "difficulty": null
      },
      {
        "id": "749d8566d2abbe6a76dd5927c261a66935e282fd4409defbfcd38a69d2a600b7",
        "domain": "numina",
        "question": "## problem statement\n\nLet $k$ be the coefficient of similarity transformation with the center at the origin. Is it true that point $A$ belongs to the image of plane $a$?\n\n$A(2; -5; 4)$\n\n$a: 5x + 2y - z + 3 = 0$\n\n$k = \\frac{4}{3}$",
        "answer": "0",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 45802,
        "difficulty": null
      },
      {
        "id": "d26ccf5330ae4be6d1bc7ed33dba5823b6034404209b36bdfb79fc5c126ec1f1",
        "domain": "numina",
        "question": "10. Given a sequence $\\left\\{a_{n}\\right\\}$ with 100 terms, satisfying $a_{1}=0, a_{100}=475$,\nand $\\left|a_{k+1}-a_{k}\\right|=5(k=1,2, \\cdots, 99)$.\nThen the number of different sequences that meet the conditions is $\\qquad$.",
        "answer": "4851",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144284,
        "difficulty": null
      },
      {
        "id": "e2d7953611464765a7288146d5067971c2ec02c4167080617cb112d2448d599d",
        "domain": "numina",
        "question": "10. (12 points) 1 kilogram of soybeans can be made into 3 kilograms of tofu, and 1 kilogram of soybean oil requires 6 kilograms of soybeans. Tofu sells for 3 yuan per kilogram, and soybean oil sells for 15 yuan per kilogram. A batch of soybeans weighs 460 kilograms, and after being made into tofu or soybean oil and sold, it yields 1800 yuan. In this batch, $\\qquad$ kilograms of soybeans were made into soybean oil.",
        "answer": "360",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128312,
        "difficulty": null
      },
      {
        "id": "286ac262da2593f2fd47c065610b2348d8c0af25e33cf2a86d7635aec0e7b44d",
        "domain": "numina",
        "question": "Folklore\n\nPetya wants to make an unusual die, which, as usual, should have the shape of a cube, with dots drawn on its faces (different numbers of dots on different faces), but at the same time, on any two adjacent faces, the number of dots must differ by at least two (at the same time, it is allowed for some faces to have more than six dots). How many dots in total are needed to be drawn for this?",
        "answer": "27",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 66239,
        "difficulty": null
      },
      {
        "id": "122417fd61f12b2e8aa5a7705ea4e9cffe48a6d2eb25410dfc4b928ea764b28d",
        "domain": "numina",
        "question": "Pechkovskiy A.n.\n\nOn a chessboard of size $8 \\times 8$, 64 points are marked - the centers of all cells. Is it possible to separate all the points from each other by drawing 13 lines that do not pass through these points?",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58353,
        "difficulty": null
      },
      {
        "id": "06a001656b958ae21bdf48734d985405bb6fdd5cdcb60db713b8844f2a2af7a3",
        "domain": "numina",
        "question": "4. At Ivanov's house, there stands an odd three-digit number. The digit in the tens place is twice as large as the digit in the units place. The digit in the hundreds place is the sum of the digits in the tens and units places. Determine all possibilities for Ivanov's house number that is divisible by 9.",
        "answer": "963",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86802,
        "difficulty": null
      },
      {
        "id": "f846ab799fad943c7624ef6864053b1b7fb9233d968788e7375208b72d266c1f",
        "domain": "numina",
        "question": "Example 7. Calculate the integral $I=\\int_{\\overparen{A B}}\\left(x^{2}-2 x y\\right) d x+\\left(y^{2}-\\right.$ $-2 x y) d y$, where $\\overparen{A B}$ is the arc of a parabola connecting points $A(-1,1)$ and $B(1,1)$ and passing through the origin.",
        "answer": "-\\frac{14}{15}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31291,
        "difficulty": null
      },
      {
        "id": "11b076eda48c536331bb9da69477ea00c0d1f36bb55261ebe7d77943b63d0369",
        "domain": "numina",
        "question": "6. (10 points) Given that the natural number $n$ has 10 divisors, $2 n$ has 20 divisors, and $3 n$ has 15 divisors, then $6 n$ has $\\qquad$ divisors.",
        "answer": "30",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 125745,
        "difficulty": null
      },
      {
        "id": "3e2c25c09ed8088fac3d53b0b03922017722fdb4e9653ebbe75ecc76c11fa3da",
        "domain": "numina",
        "question": "5. The cross-section of a regular hexagonal pyramid SABCDEF is formed by a plane passing through the center of the base $A B C D E F$ and parallel to the median $C M$ of the lateral face $S C D$ and the apothem $S N$ of the lateral face $S A F$. The side of the base of the pyramid is 8, and the distance from the vertex $S$ to the cutting plane is $3 \\sqrt{13 / 7}$. Find the cosine of the angle between the cutting plane and the base plane. (20 points)",
        "answer": "\\frac{\\sqrt{3}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10540,
        "difficulty": null
      },
      {
        "id": "7bf8acd7431668dd4a0877977f1edd3db66c17ead574d39770700a2b524257f5",
        "domain": "numina",
        "question": "6. Let $a_{0}=-2, b_{0}=1$, and for $n \\geq 0$, let\n$$\n\\begin{array}{l}\na_{n+1}=a_{n}+b_{n}+\\sqrt{a_{n}^{2}+b_{n}^{2}}, \\\\\nb_{n+1}=a_{n}+b_{n}-\\sqrt{a_{n}^{2}+b_{n}^{2}} .\n\\end{array}\n$$\n\nFind $a_{2012}$.",
        "answer": "2^{1006}\\sqrt{2^{2010}+2}-2^{2011}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151405,
        "difficulty": null
      },
      {
        "id": "057e35451288fb5fa71d4d055ca09e20da5778042211930651d647b9da7aadd9",
        "domain": "numina",
        "question": "39. (5 points)\nFour pirates are dividing 15 identical gold coins. The first pirate must get at least 4 coins, the second pirate can get none, the third pirate must get at least 2 coins, and the fourth pirate must get at least 1 coin. There are $\\qquad$ different ways to divide the coins.",
        "answer": "165",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 113010,
        "difficulty": null
      },
      {
        "id": "102e5bc0eb2e7b99ca7506c070b0b521dd84d3ed6239c523ac80c79479710575",
        "domain": "numina",
        "question": "8. The URL of the Hua Cup Competition is www.huabeisai.cn, and the letters form the following equation:\nwww+hua+bei+sai+cn=2008.\n\nIf each letter represents a digit from $0 \\sim 9$, the same letter represents the same digit, different letters represent different digits, and $\\mathrm{w}=8, \\mathrm{~h}=6, \\mathrm{a}=9, \\mathrm{c}=7$, then the minimum value of the three-digit number $\\overline{b e} i$ is _. $\\qquad$",
        "answer": "103",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 114038,
        "difficulty": null
      },
      {
        "id": "42a397e707a50e84a6cf28444fa928d82b305092fcf0079b90c86b517f5c29b3",
        "domain": "numina",
        "question": "7. [4] My friend and I are playing a game with the following rules: If one of us says an integer $n$, the opponent then says an integer of their choice between $2 n$ and $3 n$, inclusive. Whoever first says 2007 or greater loses the game, and their opponent wins. I must begin the game by saying a positive integer less than 10 . With how many of them can I guarantee a win?",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 150862,
        "difficulty": null
      },
      {
        "id": "c804fb79d1d128427728fdccfcafdaeb3e083aa4c07fa94d39c9af529847f5f2",
        "domain": "numina",
        "question": "5. Divide $1,2,3,4,5,6,7,8,9$ into 3 groups, each containing three numbers, then the probability that the sum of the numbers in each of these three groups is a prime number is",
        "answer": "\\frac{1}{140}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 142345,
        "difficulty": null
      },
      {
        "id": "45a8aea32a5e97695dbabbd6c25782ab965ba3bf3038630dae176e0050f7cff2",
        "domain": "numina",
        "question": "18. a) Into how many parts can a plane be divided by two closed curves, one of which is a circle and the other is the boundary of a square?\n\nb) $* *$ Into how many parts can space be divided by two closed surfaces, one of which is a sphere and the other is the surface of a cube?",
        "answer": "16",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 23783,
        "difficulty": null
      },
      {
        "id": "17a6e254f6dca11d0f1d340c1e91237436857215f74c6aa9c6ef90d19927e18c",
        "domain": "numina",
        "question": "6-101. Let $x, y$ be positive numbers, and $S$ be the smallest number among $x, y+\\frac{1}{x}, \\frac{1}{y}$. Find the possible maximum value of $S$, and the values of $x, y$ that achieve this maximum value.",
        "answer": "\\sqrt{2}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 107135,
        "difficulty": null
      },
      {
        "id": "520266a325905879f667475305c6a594afc1fb7275830600d3439d221b9f1730",
        "domain": "numina",
        "question": "Pooolish A.\n\nChichikov is playing with Nozdryov. First, Nozdryov distributes 1001 nuts into three boxes. After looking at the distribution, Chichikov names any integer \\( N \\) from 1 to 1001. Then Nozdryov must, if necessary, move one or several nuts to an empty fourth box and present Chichikov with one or several boxes, where the total number of nuts is exactly \\( N \\). As a result, Chichikov will receive as many dead souls as the number of nuts Nozdryov moved. What is the maximum number of souls Chichikov can guarantee for himself, regardless of how Nozdryov plays?",
        "answer": "71",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 71603,
        "difficulty": null
      },
      {
        "id": "820f5b5180fb04196a22e114e080ced3452a1ec43c995d9cd18bc2f10e65687f",
        "domain": "numina",
        "question": "2.044. $\\left(\\frac{\\sqrt{x-a}}{\\sqrt{x+a}+\\sqrt{x-a}}+\\frac{x-a}{\\sqrt{x^{2}-a^{2}}-x+a}\\right): \\sqrt{\\frac{x^{2}}{a^{2}}-1} ; x>a>0$.\n\n2.044. $\\left(\\frac{\\sqrt{x-a}}{\\sqrt{x+a}+\\sqrt{x-a}}+\\frac{x-a}{\\sqrt{x^{2}-a^{2}}-x+a}\\right): \\sqrt{\\frac{x^{2}}{a^{2}}-1} ; x>a>0$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50411,
        "difficulty": null
      },
      {
        "id": "0f9a7710079fb331960d90a4d3737e7856c07ee7732dd85c462f48637adbc1ba",
        "domain": "numina",
        "question": "10. (20 points) Through the focus $F$ of the parabola $\\Gamma: x^{2}=4 y$, draw two different lines $l_{1}$ and $l_{2}$ with slopes $k_{1}$ and $k_{2}$ respectively, and $k_{1}+k_{2}=2$. Line $l_{1}$ intersects the parabola $\\Gamma$ at points $A$ and $B$, and line $l_{2}$ intersects the parabola $\\Gamma$ at points $C$ and $D$. The circles $\\odot M$ and $\\odot N$ have diameters $AB$ and $CD$ respectively. The line containing the common chord of these circles is denoted as $l$. Find the minimum distance from point $M$ to the line $l$.",
        "answer": "\\frac{3\\sqrt{5}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 92932,
        "difficulty": null
      },
      {
        "id": "ff15e3ead824b5113bcfd4e1d150b65d613f555ab57abacc9ac44d1a7984b969",
        "domain": "numina",
        "question": "A Barber (friseur) from Béts moved to Debretzen to continue his trade; but he became impoverished within 3 years, although he had some money. In the first year, he was forced to spend half of his money, in the second year, a third of what he had brought; in the third year, 200 forints, when he only had $50 \\mathrm{Ft}$ left for the return trip: - how many forints did he have when he moved; and how much did he spend each year?",
        "answer": "1500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 175228,
        "difficulty": null
      },
      {
        "id": "06b01ea54c018fde964f6406975c05a8a95b30e5475ff254ee8369ed434608a7",
        "domain": "numina",
        "question": "4.4. To the fraction $\\frac{1}{6}$, some fraction was added, and the result turned out to be a proper fraction with a denominator less than 8. What is the largest fraction that could have been added?",
        "answer": "\\frac{29}{42}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13747,
        "difficulty": null
      },
      {
        "id": "8c00fef1c3df6532bbaced41d7051b35e0fb1b58e9fe02230cfc268ad271d523",
        "domain": "numina",
        "question": "(2) Let $a<-1$, and the variable $x$ satisfies $x^{2}+a x \\leqslant-x$, and the minimum value of $x^{2}+a x$ is $-\\frac{1}{2}$, then $a=$ $\\qquad$.",
        "answer": "-\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122381,
        "difficulty": null
      },
      {
        "id": "ff246f1fce93d72374d2c5e94e3f0af7c227235c99231cb892c61897e6b149b5",
        "domain": "numina",
        "question": "14. In the quadrilateral pyramid $P-ABCD$, the side face $PAD$ is an equilateral triangle, and it is perpendicular to the base $ABCD$, the base $ABCD$ is a rhombus with a side length of 2, $\\angle BAD=60^{\\circ}$, $N$ is the midpoint of $PB$, and the section $DAN$ intersects $PC$ at $M$. Find the size of the dihedral angle $P-AD-M$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102871,
        "difficulty": null
      },
      {
        "id": "ab2b7c2b1b6d20f0d4c5a20c8f5cab3dd95f31fb324e795cb0fe67911a8d6058",
        "domain": "numina",
        "question": "6. Next to the number 2022 on the board, an unknown positive number less than 2022 was written. Then one of the numbers on the board was replaced by their arithmetic mean. This replacement was performed 9 more times, and the arithmetic mean was always an integer. Find the smaller of the numbers originally written on the board.",
        "answer": "998",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15875,
        "difficulty": null
      },
      {
        "id": "a71cff0f2e05ba336ce8473b3998d63e668a8b9d92d495d9c98ae3344c43ec89",
        "domain": "numina",
        "question": "$9.0 .18 \\times 0.81+0.18+0.81=(\\quad)$ .\n\n\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly. \n\nHowever, the provided text is already in a mathematical expression format and does not contain any non-English words or phrases to translate. The expression remains the same in English:\n\n$9.0 .18 \\times 0.81+0.18+0.81=(\\quad)$ .",
        "answer": "\\frac{139}{121}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143792,
        "difficulty": null
      },
      {
        "id": "5b9c67d42f66b7c1080741f62d88009bf9ecf27947924398c286f34521a36c46",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{(2 n+1)^{3}-(2 n+3)^{3}}{(2 n+1)^{2}+(2 n+3)^{2}}$",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47110,
        "difficulty": null
      },
      {
        "id": "6eb0204aa1c71109eaa32ec5be8ee961ba9a71d9c6f9e4e29d2bfbf113f92a99",
        "domain": "numina",
        "question": "9.2. In a kindergarten, each child was given three cards, each of which had either \"MA\" or \"NYA\" written on it. It turned out that 20 children could form the word \"MAMA\" from their cards, 30 children could form the word \"NYANYA,\" and 40 children could form the word \"MANYA.\" How many children had all three cards the same?",
        "answer": "10",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9641,
        "difficulty": null
      },
      {
        "id": "0b86c682bf5ae081c97adc0f5a10c8b3dcbae441af05295b6e49fc0645749cb0",
        "domain": "numina",
        "question": "A point $D$ is chosen on side $B C$ of triangle $A B C$. The line tangent to the incircles of triangles $A B D$ and $A D C$ and different from $B C$ and $A D$ intersects segment $A D$ at $T$. If $A B=40 \\text{ cm}, A C=50 \\text{ cm}$, and $B C=60 \\text{ cm}$, determine the value of the length of $A T$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4a19aea8e08094ae376fg-07.jpg?height=539&width=1108&top_left_y=1079&top_left_x=546)",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74578,
        "difficulty": null
      },
      {
        "id": "54522e5260ab489daaa86c60cfd6a02c5007d28bf399d010d93167c6f48d70d1",
        "domain": "numina",
        "question": "Example 12 A deck of tri-color cards consists of 32 cards in total, with 10 cards of each color: red, yellow, and blue, numbered $1,2, \\cdots, 10$; there are also a big joker and a small joker, each numbered 0. Several cards are drawn from this deck, and their scores are calculated according to the following rule: each card numbered $k$ is worth $2^{k}$ points. If the total score of these cards is 2004, they are called a \"good\" card group. Find the number of \"good\" card groups.\n(2004 National Girls' Olympiad)",
        "answer": "1006009",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 104357,
        "difficulty": null
      },
      {
        "id": "c66fc98d4a508286154e046ec5a27423044b32b8a152e24b869c02c70706737c",
        "domain": "numina",
        "question": "\\section*{Exercise 2 - 330932}\n\nFor each positive integer \\(n\\), imagine another number \\(n^{\\prime}\\) formed according to the following rule: From the digit representation of \\(n\\) in the decimal system, the first digit is removed and appended to the end of the last digit.\n\nThen let \\(n^{\\prime}\\) be the number with the resulting digit representation. Investigate whether there are numbers \\(n\\) divisible by 7 for which \\(n^{\\prime}=n / 7\\) holds!",
        "answer": "10^{21}+\\frac{10^{21}-7}{69}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 184787,
        "difficulty": null
      },
      {
        "id": "0fb0e7f3fbec2a2041e4219ca06113e6a7d243fdf11da9834b6b15030ea95533",
        "domain": "numina",
        "question": "33. For $n=1,2,3, \\ldots$, let $a_{n}=n^{2}+100$, and let $d_{n}$ denote the greatest common divisor of $a_{n}$ and $a_{n+1}$. Find the maximum value of $d_{n}$ as $n$ ranges over all positive integers.",
        "answer": "401",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160508,
        "difficulty": null
      },
      {
        "id": "23f43fd8a14bad937fa45c186a4a9b9370cf007f8aacd90e6b0981e942c66412",
        "domain": "numina",
        "question": "4. In each of the following 7-digit natural numbers:\n1001011,5550000,3838383,7777777,\nevery digit in the number appears at least 3 times. Find the number of such 7 -digit natural numbers.",
        "answer": "2844",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160302,
        "difficulty": null
      },
      {
        "id": "e8672b6496950e319e661e60b4e9136ba46f15adb05540e24314f0375090fc0a",
        "domain": "numina",
        "question": "5. Maximum 15 points. The company \"Intelligence, Inc\" has developed a robot with artificial intelligence. To manufacture it, a special machine is required, which can produce 1 robot in 1 hour. The company owns a large number of such machines, but the created robot is so intelligent that it can produce an exact copy of itself in exactly 2 hours. \"Intelligence, Inc\" has received an urgent order for 3250 robots, which need to be manufactured in just 10 hours. The company's expenses for creating one robot (regardless of the time of its production and whether it was created by another robot or a special machine) are constant and amount to 100 monetary units, and the maintenance of one special machine during its operation to produce a robot costs the company 70 monetary units. \"Intelligence, Inc\" does not incur any other expenses for creating robots. What are the minimum costs the company will incur in fulfilling the order?",
        "answer": "328710",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 7626,
        "difficulty": null
      },
      {
        "id": "30d012699a145afe7038baee081281054b469591bc41dd781cedd7c6e9e83535",
        "domain": "numina",
        "question": "Problem 5.3. By September 1st, Vlad bought himself several ballpoint and gel pens. He noticed that if all the pens he bought were gel pens, he would have paid 4 times more than he actually did. And if all the pens were ballpoint pens, the purchase would have cost him 2 times less than the actual price. How many times more expensive is a gel pen compared to a ballpoint pen?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12867,
        "difficulty": null
      },
      {
        "id": "aed62ab1fc5aad744c2f67df80fa5800b19e8b96238fe2d604a14763f87f7721",
        "domain": "numina",
        "question": "## Problem Statement\n\nBased on the definition of the derivative, find $f^{\\prime}(0)$:\n\n$$\nf(x)=\\left\\{\\begin{array}{c}\nx+\\arcsin \\left(x^{2} \\sin \\frac{6}{x}\\right), x \\neq 0 \\\\\n0, x=0\n\\end{array}\\right.\n$$",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46673,
        "difficulty": null
      },
      {
        "id": "d016897b6ae7070403fbbc2196a93ab581b3c6347ddf6cd77521816a8f707eb5",
        "domain": "numina",
        "question": "## Task 4 - 240814\n\nFrom three congruent isosceles right triangles with given leg length $a$, a trapezoid can be formed.\n\na) Draw such a trapezoid!\n\nb) Such a trapezoid can be divided into four congruent trapezoids. Draw such a division!\n\nc) Determine the lengths of the parallel sides, the height, and the area of one of these smaller trapezoids in terms of $a$!",
        "answer": "\\frac{3}{8}^{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 184200,
        "difficulty": null
      },
      {
        "id": "ed8e071ed347de18a1505738814e0bc0f27ab2c56deddbbb6b8278b4b9e887d6",
        "domain": "numina",
        "question": "19. [11] Let $P$ be a polynomial with $P(1)=P(2)=\\cdots=P(2007)=0$ and $P(0)=2009$ !. $P(x)$ has leading coefficient 1 and degree 2008. Find the largest root of $P(x)$.",
        "answer": "4034072",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151049,
        "difficulty": null
      },
      {
        "id": "f7df7979acc8ac5a98f1a6b15c91ce94b3a7b55e4cf886759bb66cea8de0284a",
        "domain": "numina",
        "question": "A passenger left their belongings in an automated storage locker, and when they came to retrieve their items, they realized they had forgotten the number. They only remember that the numbers 23 and 37 were in the code. To open the locker, the correct five-digit number must be entered. What is the minimum number of codes that need to be tried to definitely open the locker?\n\n#",
        "answer": "356",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 54938,
        "difficulty": null
      },
      {
        "id": "4d2c0dfcb9b9d6085529012ab3dea60ef20e7e3825b7a230a22274e64cab547f",
        "domain": "numina",
        "question": "$\\begin{array}{l}\\text { II. ( } 50 \\text { points) } \\\\ \\text { Infinitely many non-degenerate } \\triangle \\text {, summing the shortest side, the second longest side, and the longest side of each } \\\\ \\text { to form } 1 \\text { triangle, what is the minimum value of the cosine of the largest angle of the resulting triangle? }\\end{array}$",
        "answer": "1-\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129850,
        "difficulty": null
      },
      {
        "id": "6a5dae094a0e256ebdfc97931502636936cff8e3791959fcaedb3655893adf3d",
        "domain": "numina",
        "question": "13.196. Two hours after leaving the factory, the driver looked at the speedometer and noticed that he had only traveled 112 km. He mentally calculated that if he continued at the same speed, he would be 30 minutes late in delivering the cargo to the station. Therefore, the driver increased his speed and arrived at the station even 30 minutes early. Determine the initial and subsequent speeds of the car, if the distance from the factory to the station according to the speedometer is 280 km.",
        "answer": "56",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51679,
        "difficulty": null
      },
      {
        "id": "ae6eaf56b80520219986c85a78252d36fa3a39bcae9138e16b6fa1fccbd5a4c2",
        "domain": "numina",
        "question": "8 The line $\\sqrt{3} x-y-\\sqrt{3}=0$ intersects the parabola $y^{2}=4 x$ at points $A$ and $B$ ($A$ is above the x-axis), and intersects the x-axis at point $F$. If $\\overrightarrow{O F}=\\lambda \\overrightarrow{O A}+\\mu \\overrightarrow{O B}$, then $\\mu^{2}-$ $\\lambda^{2}=$ $\\qquad$ .",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 93640,
        "difficulty": null
      },
      {
        "id": "af659c9a985f017dcade041024f44b984ade7bba9a93e0a005688f2ddbcbf5d8",
        "domain": "numina",
        "question": "4. Throughout the year, Marko took several exams and scored a certain number of points on them. If Marko were to score 89 points on his next exam, his average score on the exams would be 91. However, if he were to score 64 points on the next exam, his average score on the exams would then be 86. How many exams has Marko taken so far?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87141,
        "difficulty": null
      },
      {
        "id": "9051715f45934552831918b207e5d7140e74e755b016545d83158242c47c350c",
        "domain": "numina",
        "question": "3. Solve the equation\n\n$$\n\\sqrt{\\frac{x-3}{11}}+\\sqrt{\\frac{x-4}{10}}+\\sqrt{\\frac{x-5}{9}}=\\sqrt{\\frac{x-11}{3}}+\\sqrt{\\frac{x-10}{4}}+\\sqrt{\\frac{x-9}{5}}\n$$",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4395,
        "difficulty": null
      },
      {
        "id": "c245d97d305781ab6860871f96adfc32780c1abe5b8bb774796def2f9883d805",
        "domain": "numina",
        "question": "Bogdanov I.I.\n\nThe Tsar summoned two sages. He gave the first 100 blank cards and ordered him to write a positive number on each (the numbers do not have to be different), without showing them to the second. Then the first can tell the second several different numbers, each of which is either written on some card or equal to the sum of numbers on some cards (without specifying how each number was obtained). The second must determine which 100 numbers are written on the cards. If he cannot, both will have their heads cut off; otherwise, a number of hairs will be plucked from each beard equal to the number of numbers the first told the second. How can the sages, without communicating, stay alive and lose the minimum number of hairs?",
        "answer": "101",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 52740,
        "difficulty": null
      },
      {
        "id": "d8219ecc526c9e0d5960c4d08419f1d4ff29586e211476a0d0b6a38d08e5551d",
        "domain": "numina",
        "question": "## Task Condition\n\nApproximately calculate using the differential.\n\n$y=x^{7}, x=2,002$",
        "answer": "128.896",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47038,
        "difficulty": null
      },
      {
        "id": "2031b21a1351be5ce5692c002b3cbbc1fc7738d5aae93ac55d255d8eb10d5bd2",
        "domain": "numina",
        "question": "6. (8 points) On the board, 34 ones are written. Every minute, Karlson erases two arbitrary numbers and writes their sum on the board, and then eats a number of candies equal to the product of the two erased numbers. What is the maximum number of candies he could have eaten in 34 minutes?",
        "answer": "561",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19644,
        "difficulty": null
      },
      {
        "id": "92c579f27d96e64f1a53b73e6bb31cb8a0b72df1943c14479265529f4bde8637",
        "domain": "numina",
        "question": "1. If the solution set of the inequality $|x+a|<b$ with respect to $x$ is $\\{x \\mid 2<x<4\\}$, then the value of $a b$ is $\\qquad$",
        "answer": "-3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 129699,
        "difficulty": null
      },
      {
        "id": "a527d8c84bcb99036bd5e6080c41bd9c7232f0712e177e318d86c8c6f6a134a8",
        "domain": "numina",
        "question": "[ Cutting into parts with special properties ]\n\nFor what smallest $n$ can a square $n \\times n$ be cut into squares $40 \\times 40$ and $49 \\times 49$ so that squares of both types are present?\n\n#",
        "answer": "2000",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66875,
        "difficulty": null
      },
      {
        "id": "a0b0ff844db6392b505c31555b7f942502ccfe96cfb3d940a1c58b8aac50aa19",
        "domain": "numina",
        "question": "A jar contains 6 crayons, of which 3 are red, 2 are blue, and 1 is green. Jakob reaches into the jar and randomly removes 2 of the crayons. What is the probability that both of these crayons are red?",
        "answer": "\\frac{1}{5}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 193988,
        "difficulty": null
      },
      {
        "id": "55708eae5b9938ee2bc1b1f2c0e2a2202392645ff8d65cba52c857d11bf85392",
        "domain": "numina",
        "question": "21 Find the integer part of\n$$\n\\frac{1}{\\frac{1}{2003}+\\frac{1}{2004}+\\frac{1}{2005}+\\frac{1}{2006}+\\frac{1}{2007}+\\frac{1}{2008}+\\frac{1}{2009}} .\n$$",
        "answer": "286",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160451,
        "difficulty": null
      },
      {
        "id": "2daaed6e8f3a0247f72c2a592e6dd490bfdd1aecf1fd4d90934c023f98190d8c",
        "domain": "numina",
        "question": "6.032. $\\sqrt{x+\\sqrt{x+11}}+\\sqrt{x-\\sqrt{x+11}}=4$.\n\n6.032. $\\sqrt{x+\\sqrt{x+11}}+\\sqrt{x-\\sqrt{x+11}}=4$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50701,
        "difficulty": null
      },
      {
        "id": "1bb25142f4eb0a810014c12aaf8477cac085bf215941f5104ff70d4968328710",
        "domain": "numina",
        "question": "9. The common ratio of the geometric sequence $a+\\log _{2} 3, a+\\log _{4} 3, a+\\log _{8} 3$ is",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109109,
        "difficulty": null
      },
      {
        "id": "a7b051d61d5058ab386a1cb19ca8ee099d040c639d70993a8e3f2e17d424cabd",
        "domain": "numina",
        "question": "Shapovalov A.V.\n\nA paper rectangular triangle was folded along a straight line so that the vertex of the right angle coincided with another vertex.\n\na) In what ratio are the diagonals of the resulting quadrilateral divided by their point of intersection?\n\nb) The resulting quadrilateral was cut along the diagonal coming out of the third vertex of the original triangle. Find the area of the smallest piece of paper formed.",
        "answer": "\\frac{1}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65141,
        "difficulty": null
      },
      {
        "id": "658a2450b9002c6168ef225d71f522cb2929d7cd0289fe349202d7ed9285bd64",
        "domain": "numina",
        "question": "14th USAMO 1985 Problem 5 0 < a 1 ≤ a 2 ≤ a 3 ≤ ... is an unbounded sequence of integers. Let b n = m if a m is the first member of the sequence to equal or exceed n. Given that a 19 = 85, what is the maximum possible value of a 1 + a 2 + ... + a 19 + b 1 + b 2 + ... + b 85 ?",
        "answer": "1700",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147971,
        "difficulty": null
      },
      {
        "id": "e9079a9f0082e25c82114bc844131afbff98b0ce2decd8afb1fa34dc72032113",
        "domain": "numina",
        "question": "Aerith bakes some cookes. On the first day, she gives away 1 cookie and then $1 / 8$ of the remaining cookies; on the second day, she gives away 2 cookies and then $1 / 8$ of the remaining cookies, and so on. On the 7th day, she gives away 7 cookies and then there are none left. How many cookies did she bake?",
        "answer": "49",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90241,
        "difficulty": null
      },
      {
        "id": "4ea75c95ddf09dcc55b55129eca10b4e7af09d26c8a1fd99123502a69df9085f",
        "domain": "numina",
        "question": "Exercise 4. A group of 20 dwarves coming out of the mine sit around a round table to count the gold nuggets each has mined. They make the following observations:\n\n$\\triangleright$ The difference in the number of nuggets between two neighboring dwarves is always 2 or 3.\n\n$\\triangleright$ All the dwarves have mined a different number of nuggets.\n\nWhat is the greatest possible difference in nuggets between the dwarf who found the most and the one who found the least?",
        "answer": "29",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 78179,
        "difficulty": null
      },
      {
        "id": "81b28f43fa8f2a811559a895a534f3530c02a48f656bd4bca2fe69cd75b48574",
        "domain": "numina",
        "question": "10. (10 points) Simplify each of the fractions $\\frac{1}{2014}, \\frac{2}{2014}, \\cdots, \\frac{2012}{2014}, \\frac{2013}{2014}$ to their simplest form, and find the sum of all the fractions that still have 2014 as their denominator after simplification.",
        "answer": "468",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105199,
        "difficulty": null
      },
      {
        "id": "755f8160588c25b86fe96d4084b729a63e9ed83bbf176c2d7e170b1bff5339ed",
        "domain": "numina",
        "question": "8. The equation $5 z^{4}+10 z^{3}+10 z^{2}+5 z+1=0$ has four complex roots, the real part of the complex root with the largest modulus is $\\qquad$ _.",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126519,
        "difficulty": null
      },
      {
        "id": "ce117f06a7f98db07ee077484c67bc68154c06b63ff81c3fc55b999c586a7e79",
        "domain": "numina",
        "question": "6. A bag contains $m$ red balls and $n$ white balls, $m>n \\geq 4$. Now, two balls are drawn at random. If the probability of drawing two balls of the same color is equal to the probability of drawing two balls of different colors, then the number of arrays $(m, n)$ that satisfy the relation $m+n \\leq 40$ is $\\qquad$ .",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143219,
        "difficulty": null
      },
      {
        "id": "ed3db7c8a79b53975a499be44f27c4ee1a3e1987cbaa4787b9dce5d53730c04a",
        "domain": "numina",
        "question": "Through every point $M$ of the plane, two hyperbolas can be drawn, whose equations, referred to the rectangular coordinate axes $O x, O y$, are\n\n$$\na^{2} x y+a y+x=0\n$$\n\nwhere $a$ is a variable parameter; on which curve $(C)$ must the point $M$ lie so that the hyperbolas intersect at right angles at this point?\n\nHow many points does the curve $(C)$ intersect the hyperbolas with the equation $a^{2} x y+a y+x=0$. Only those points should be considered which are neither at infinity nor coincide with the origin of the coordinate system.\n\nWhat relationship must hold between $a$ and $b$ so that the following two hyperbolas, whose equations are\n\n$$\n\\begin{aligned}\n& a^{2} x y+a y+x=0 \\\\\n& b^{2} x y+b y+x=0\n\\end{aligned}\n$$\n\nintersect at right angles at a point different from the origin of the coordinate system. This relationship defines a curve $(C')$; the curve must be constructed. How many finite and non-origin points does the previously defined curve $(C)$ intersect the curve $(C')$?\n\n(Licentiatus stipendium competition written paper, 1894.)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 176188,
        "difficulty": null
      },
      {
        "id": "6f3e5b90fc2c62e6b15b4ca5c0bc86855dcfa61debcf5ad06ccbd94fe1e3716f",
        "domain": "numina",
        "question": "16. (1990 IMO Japan First Round Selection Test) Let $x, y, z > 0$, and $x + y + z = 1$, find the minimum value of $\\frac{1}{x} + \\frac{4}{y} + \\frac{9}{z}$.",
        "answer": "36",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 99947,
        "difficulty": null
      },
      {
        "id": "6da64c8a4f51066d1016c7609eece0d3ef689acbb4fee5d51c51075529b4b130",
        "domain": "numina",
        "question": "33. Today is January 18, 2020, and it is displayed on the electronic calendar as 20200118. If there is a multi-digit number $202001188462 \\cdots \\cdots$, starting from the 9th digit, each digit is the unit digit of the product of the three preceding digits. According to this rule, what is the 2020th digit of this multi-digit number? $\\qquad$ .",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100047,
        "difficulty": null
      },
      {
        "id": "12ce739c4126341ccb98c3f933c269fd944d8d1939f07687481821f4f9330ac0",
        "domain": "numina",
        "question": "5. The number of 4-digit numbers whose sum of digits equals 12 is. $\\qquad$",
        "answer": "342",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 138119,
        "difficulty": null
      },
      {
        "id": "c0ea26a56c20cbfc49ed50af814d657cedd9786a4e1a2d4f9c1d757610f529a4",
        "domain": "numina",
        "question": "7. (10 points) 6 small circles of equal radius and 1 large circle are arranged as shown in the figure. The area of the large circle is 120. Therefore, the area of one \n\nsmall circle is $\\qquad$ .",
        "answer": "40",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 134413,
        "difficulty": null
      },
      {
        "id": "a78481a35f12dc030863caf31ce36cfebfcc3decf7f9ea80a57ed608a1d64992",
        "domain": "numina",
        "question": "1. (6 points) Calculate: $\\frac{(2015-201.5-20.15)}{2.015}=$",
        "answer": "890",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 126459,
        "difficulty": null
      },
      {
        "id": "e48a5dcd1dc8e7f4419727415ed51396e3c5354ca9b8f3738ed4999f91e32dc0",
        "domain": "numina",
        "question": "2. (7 points) The kids were given the task to convert the turtle's speed from centimeters per second to meters per minute. Masha got an answer of 25 m/min, but she thought there were 60 cm in a meter and 100 seconds in a minute. Help Masha find the correct answer.",
        "answer": "9",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16753,
        "difficulty": null
      },
      {
        "id": "ae01732ea5490f046a62090bd91031f572cee1ad89990338fa877a138e019074",
        "domain": "numina",
        "question": "5. On the side $\\overline{A B}$ of the rectangle $A B C D$, several points have been chosen, on the sides $\\overline{B C}$ and $\\overline{D A}$ four points each, and on the side $\\overline{C D}$ five points. None of the chosen points is a vertex of the given rectangle. Using the chosen points as vertices, exactly 4040 quadrilaterals can be drawn such that at most one vertex lies on the segment $\\overline{A B}$. How many points were chosen on the side $\\overline{A B}$?",
        "answer": "13",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89968,
        "difficulty": null
      },
      {
        "id": "d66bba6b77a5d910d0e79dfba8744bb8ec909da6bccfe4f4bd60e018a98ce966",
        "domain": "numina",
        "question": "1.041. $\\frac{\\left(4-3.5 \\cdot\\left(2 \\frac{1}{7}-1 \\frac{1}{5}\\right)\\right): 0.16}{X}=\\frac{3 \\frac{2}{7}-\\frac{3}{14}: \\frac{1}{6}}{41 \\frac{23}{84}-40 \\frac{49}{60}}$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50364,
        "difficulty": null
      },
      {
        "id": "55e9a59dd7b33a9936c1dc171bbb8f51e658bbceaa2ead4c2c8789ede331db56",
        "domain": "numina",
        "question": "Example 11 For real number $x$, find the maximum value of the function $f(x)=$ $\\sqrt{8 x-x^{2}}-\\sqrt{14 x-x^{2}-48}$.",
        "answer": "2\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102776,
        "difficulty": null
      },
      {
        "id": "d3433d894c19a20fd7b3db01755f9e63b2796663a0a1141c69b5c04060524ed4",
        "domain": "numina",
        "question": "## Task B-4.2.\n\nDetermine the natural number $n$, so that the ratio of the seventh term counting from the beginning to the seventh term counting from the end, in the expansion of the binomial $\\left(\\sqrt[3]{2}+\\frac{1}{\\sqrt[3]{3}}\\right)^{n}$, is equal to $\\frac{1}{6}$.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88660,
        "difficulty": null
      },
      {
        "id": "497b49a9dbd1a4690f7a0918e036a45bf326d4dda1c73cf0b3b9da3140860ef2",
        "domain": "numina",
        "question": "5. (8 points) Multiply 4.02 by a two-digit integer, and the product is an integer. Then, 10 times this product is $\\qquad$ .",
        "answer": "2010",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113328,
        "difficulty": null
      },
      {
        "id": "cc2278b51aeb41c8dd82c5061fb81104195de9e42288a55e0ec296a4c9c053e1",
        "domain": "numina",
        "question": "6. Calculate with an accuracy of one-tenth the value of the expression $\\sqrt{86+41 \\sqrt{86+41 \\sqrt{86+\\ldots}}}$",
        "answer": "43",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8590,
        "difficulty": null
      },
      {
        "id": "bdbbd32d9c65692f5ac671377b7cb3896871802eb039b7a6e2d24191b09c3ec7",
        "domain": "numina",
        "question": "9. Let $O, O_{1}, O_{2}, O_{3}, O_{4}$ be points such that $O_{1}, O, O_{3}$ and $O_{2}, O, O_{4}$ are collinear in that order, $O O_{1}=$ $1, O O_{2}=2, O O_{3}=\\sqrt{2}, O O_{4}=2$, and $\\measuredangle O_{1} O O_{2}=45^{\\circ}$. Let $\\omega_{1}, \\omega_{2}, \\omega_{3}, \\omega_{4}$ be the circles with respective centers $O_{1}, O_{2}, O_{3}, O_{4}$ that go through $O$. Let $A$ be the intersection of $\\omega_{1}$ and $\\omega_{2}, B$ be the intersection of $\\omega_{2}$ and $\\omega_{3}, C$ be the intersection of $\\omega_{3}$ and $\\omega_{4}$, and $D$ be the intersection of $\\omega_{4}$ and $\\omega_{1}$, with $A, B, C, D$ all distinct from $O$. What is the largest possible area of a convex quadrilateral $P_{1} P_{2} P_{3} P_{4}$ such that $P_{i}$ lies on $O_{i}$ and that $A, B, C, D$ all lie on its perimeter?",
        "answer": "8+4\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151415,
        "difficulty": null
      },
      {
        "id": "f8b9b8c46d21eadb95f04932b698c2850222e5068bf4d7c9e26b87fa13e0b4c4",
        "domain": "numina",
        "question": "Shen A.H.\n\nThere is an infinite one-way strip of cells, numbered with natural numbers, and a bag with ten stones. Initially, there are no stones in the cells of the strip. The following actions can be performed:\n\n- move a stone from the bag to the first cell of the strip or vice versa;\n- if there is a stone in the cell with number \\$\\$, then a stone can be moved from the bag to the cell with number $\\$ \\mathrm{i}+1 \\$$ or vice versa.\n\nIs it possible, acting according to these rules, to place a stone in the cell with number 1000?",
        "answer": "1023",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56685,
        "difficulty": null
      },
      {
        "id": "11779f6d863863dda9ff80720f665906cd2baa2bdc3fbd9633fb8e5d5918c3b4",
        "domain": "numina",
        "question": "Example 13. Find the flux of the vector field $\\mathbf{v}=(x-2 y+1) \\mathbf{i}+(2 x+ y-3 z) \\mathbf{j}+(3 y+z) \\mathbf{k}$ through the part of the sphere $x^{2}+y^{2}+z^{2}$, located in the first octant, into the region where $x^{2}+y^{2}+z^{2}>1$.",
        "answer": "\\frac{3}{4}\\pi",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31615,
        "difficulty": null
      },
      {
        "id": "79a6d899905e56e6c36609eff95f4eccd2b210c7572d1ab2e1e5979b0f948409",
        "domain": "numina",
        "question": "Problem 3. In a certain company, there are 100 shareholders, and any 66 of them collectively own at least $50\\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own? (The percentage of shares in the company owned by a shareholder can be non-integer.)",
        "answer": "25",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 4510,
        "difficulty": null
      },
      {
        "id": "3179fb1a8d89db2b1e30f94cf5c5d8aa4b6c36e06265f68f24da1ebe71ab7e6b",
        "domain": "numina",
        "question": "1st ASU 1967 Problem 9 ABCD is a unit square. One vertex of a rhombus lies on side AB, another on side BC, and a third on side AD. Find the area of the set of all possible locations for the fourth vertex of the rhombus. Solution",
        "answer": "2\\frac{1}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 148726,
        "difficulty": null
      },
      {
        "id": "8b5f752e8fb037a89585aafb00134d4b183bb4c773e78ea36dab6d4e475c479e",
        "domain": "numina",
        "question": "I4.4 Find the value of $1+3^{c}+\\left(3^{c}\\right)^{2}+\\left(3^{c}\\right)^{3}+\\left(3^{c}\\right)^{4}$ divided by $1+3+3^{2}+3^{3}+3^{4}$, denoted as $d$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153411,
        "difficulty": null
      },
      {
        "id": "7062bd4eaa0191f3b533aa0a8c807251558203d1faf1b83d60ef0b868d56afb0",
        "domain": "numina",
        "question": "G1.3 Given that $\\tan \\alpha=-\\frac{1}{2}$. If $c=\\frac{2 \\cos \\alpha-\\sin \\alpha}{\\sin \\alpha+\\cos \\alpha}$, find the value of $c$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153047,
        "difficulty": null
      },
      {
        "id": "fe1a2ac3ab6606c59d52f4ca85fa3611e57ffa0ee699853327a572673025d7f6",
        "domain": "numina",
        "question": "1. In the cube $A B C D-A_{1} B_{1} C_{1} D_{1}$, $O$ is the center of face $A B C D$, and $O_{1}$ is the center of face $A D D_{1} A_{1}$. Find the cosine value of the angle formed by the skew lines $D_{1} O$ and $B O_{1}$.",
        "answer": "\\frac{5}{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114274,
        "difficulty": null
      },
      {
        "id": "a4bb9db208f1e46b2af80c31adae2771ee95607602106c076d178f353cff1c5a",
        "domain": "numina",
        "question": "3. The sides $B C$ and $A D$ of a quadrilateral $A B C D$ are parallel and the diagonals intersect in $O$. For this quadrilateral $|C D|=|A O|$ and $|B C|=|O D|$ hold. Furthermore $C A$ is the angular bisector of angle $B C D$.\n\nDetermine the size of angle $A B C$.\n\nAttention: the figure is not drawn to scale.\n\nYou have to write down your reasoning step by step in text and formulas. No points will be awarded for annotations in a picture\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_e07875fe43bd53054a56g-1.jpg?height=414&width=505&top_left_y=1483&top_left_x=1392)\nalone.",
        "answer": "126",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 85770,
        "difficulty": null
      },
      {
        "id": "fdd28efe60c86783af9791175ac88dfbcade2704085d5f5632966813aa0c86f6",
        "domain": "numina",
        "question": "Example 10. At an enterprise, products of a certain type are manufactured on three production lines. The first line produces $30 \\%$ of the products from the total production volume, the second line - $25 \\%$, and the third line produces the remaining part of the products. Each line is characterized by the following percentages of good products: $97 \\%, 98 \\%, 96 \\%$. Determine the probability that a randomly selected product, produced by the enterprise, will be defective.",
        "answer": "0.032",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32973,
        "difficulty": null
      },
      {
        "id": "471eb427336b3c605077935fe9e1a30f2a12df553aab10e1c6c7bde15a8029f4",
        "domain": "numina",
        "question": "10.208. Perpendiculars are drawn from the vertex of the acute angle of a rhombus to the lines containing the sides of the rhombus to which this vertex does not belong. The length of each perpendicular is 3 cm, and the distance between their bases is $3 \\sqrt{3}$ cm. Calculate the lengths of the diagonals of the rhombus.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 48818,
        "difficulty": null
      },
      {
        "id": "655d207fa8d9647c9bc360c9ff8175ff367f28329b02fff72ea04fc72e323afc",
        "domain": "numina",
        "question": "14. Use 6 white beads, 8 black beads, and 1 red bead to string into a necklace. How many different ways are there to do this?",
        "answer": "1519",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 92819,
        "difficulty": null
      },
      {
        "id": "cf23111e5f8aac56003fa13d5b40557ebba34d1824e4b90ec4f87f8c5fad3f32",
        "domain": "numina",
        "question": "[ Unusual Constructions (Miscellaneous). ] [ Measuring the lengths of segments and measures of angles. Adjacent angles. $]$\n\nThere is an angle ruler with an angle of $70^{\\circ}$. How can you construct an angle of $40^{\\circ}$ with it?\n\n#",
        "answer": "40",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60239,
        "difficulty": null
      },
      {
        "id": "a62f3f733dae12ba01f1f83be8bedfb0e267e05fc58aec8533fc96f16bb32005",
        "domain": "numina",
        "question": "17. From a square with a side length of 20, remove a rectangle with an area of 36, where the two side lengths of the rectangle are integers, and one side of the rectangle is part of one side of the square. The maximum perimeter of the remaining shape is $\\qquad$.",
        "answer": "116",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105011,
        "difficulty": null
      },
      {
        "id": "3afe0886709f9fd0e3bbcd8341bc043cad3b0c7150b3897af7879a1ca7a42d9d",
        "domain": "numina",
        "question": "1. What is the maximum number of lattice points (i.e. points with integer coordinates) in the plane that can be contained strictly inside a circle of radius 1 ?",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152129,
        "difficulty": null
      },
      {
        "id": "9030ea6ef22f5c74fff1dc8a41838ee958fef6d303515f84eda98bafa838999b",
        "domain": "numina",
        "question": "[ Decimal numeral system ] $[\\quad$ Case enumeration $\\quad]$\n\n## Author: Raskina I.V.\n\nThe year 2009 has the property that by rearranging the digits of the number 2009, it is impossible to obtain a smaller four-digit number (numbers do not start with zero). In which year will this property reappear for the first time?",
        "answer": "2022",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68261,
        "difficulty": null
      },
      {
        "id": "ef9b31bb2f35f6b39d29c454db8158061458915f87d3efa6cfef9de9a39523a7",
        "domain": "numina",
        "question": "6. (10 points) For a natural number $N$, if at least seven of the nine natural numbers from 1 to 9 are factors of $N$, then $N$ is called a \"Seven-Star Number\". Among the natural numbers greater than 2000, the smallest \"Seven-Star Number\" is $\\qquad$",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 134937,
        "difficulty": null
      },
      {
        "id": "c26aa97e4c1b624e1ef23224666af9635ad96e17bf79f79c21151fa410995041",
        "domain": "numina",
        "question": "7, 10 consecutive four-digit numbers whose sum is a multiple of 37, the smallest sum of these 10 numbers is",
        "answer": "10175",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147278,
        "difficulty": null
      },
      {
        "id": "0530b776fbc82b8a53a6d6d25be8e69e1aa0d22977a70d9a3d629c590777969c",
        "domain": "numina",
        "question": "## SUBJECT III\n\nDetermine the measure of the angle formed by the bisectors of two consecutive angles in a parallelogram.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 178754,
        "difficulty": null
      },
      {
        "id": "597e9557e789384217cb7b5cc464c539f150dcc75b5b4cbddabc91d423ef066a",
        "domain": "numina",
        "question": "Task 3. Determine the largest five-digit number such that the product of its digits is 2520?",
        "answer": "98751",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188564,
        "difficulty": null
      },
      {
        "id": "ab7eb66f54b9e526b7489eb4126e0dd2aed22d508347d6a01a6b9420645397bf",
        "domain": "numina",
        "question": "2. Through the right focus of the hyperbola $x^{2}-\\frac{y^{2}}{2}=1$, a line $l$ intersects the hyperbola at points $A$ and $B$. If a real number $\\lambda$ makes $|A B|=$ $\\lambda$, and there are exactly 3 such lines $l$, then $\\lambda=$ $\\qquad$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 94849,
        "difficulty": null
      },
      {
        "id": "cba9ee4b91250e11323481d4e6977f118d860ae0d3304c642051a0ed47614652",
        "domain": "numina",
        "question": "The dandelion blooms in the morning, flowers yellow for two days, turns white on the third morning, and by evening it has scattered. Yesterday afternoon, there were 20 yellow and 14 white dandelions on the meadow, and today there are 15 yellow and 11 white.\n\na) How many yellow dandelions were there on the meadow the day before yesterday?\n\nb) How many white dandelions will there be on the meadow tomorrow?",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 72380,
        "difficulty": null
      },
      {
        "id": "fc7cdab87ec936fc322b0d197fbe4970c5071a396d7af9cdc1c66440d8b6ce11",
        "domain": "numina",
        "question": "II. (50 points) There are 26 stamp collectors from different countries who want to exchange the latest commemorative stamps issued by their respective countries through correspondence. To ensure that each of the 26 individuals ends up with a complete set of the latest commemorative stamps from all 26 countries, what is the minimum number of letters they need to send?",
        "answer": "50",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132142,
        "difficulty": null
      },
      {
        "id": "34e82e383e67b32e14ba81641c482ae0988c01e0de0acbbb8cc4f782ddf0a3ee",
        "domain": "numina",
        "question": "16 Let $x>0$, then the minimum value of $f(x)=\\frac{\\left(x+\\frac{1}{x}\\right)^{4}-\\left(x^{4}+\\frac{1}{x^{4}}\\right)}{\\left(x+\\frac{1}{x}\\right)^{3}-\\left(x^{3}+\\frac{1}{x^{3}}\\right)}$ is $\\qquad$ .",
        "answer": "\\frac{7}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132636,
        "difficulty": null
      },
      {
        "id": "133b2cbb581f1feffa267b9158aedcbb61c983bf567aff200781d020b1bb32ad",
        "domain": "numina",
        "question": "9. (12 points) Select 8 different digits from the numbers $1-9$ to fill in the squares below, so that the vertical equation holds. The largest possible four-digit number is $\\qquad$ .\n\n\n\nThe translation is provided while preserving the original text's line breaks and format.",
        "answer": "1769",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 125177,
        "difficulty": null
      },
      {
        "id": "bb023346be9ad82b3e9e8b7760ce92af63af33e88c0e3502adccc6cb832f9c82",
        "domain": "numina",
        "question": "641. Study the symmetric properties of an equilateral triangle.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 40338,
        "difficulty": null
      },
      {
        "id": "746cbfec112390fee3b258d80a335a521b749d649113f36bfe4b342a75c62b37",
        "domain": "numina",
        "question": "7. Given $F_{1}, F_{2}$ are the left and right foci of the hyperbola $C: \\frac{x^{2}}{4}-\\frac{y^{2}}{12}=1$, point $P$ is on the hyperbola $C$, and $G, I$ are the centroid and incenter of $\\triangle F_{1} P F_{2}$, respectively. If $G I \\| x$-axis, then the circumradius $R$ of $\\triangle F_{1} P F_{2}$ is $\\qquad$ ـ.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98200,
        "difficulty": null
      },
      {
        "id": "29e30382f61b3b0986c9d941cc1dc73e7dba1f5c2c535d881938fa18f2dcd42b",
        "domain": "numina",
        "question": "283. Find a number of the form $7 x 36 y 5$, divisible by 1375.",
        "answer": "713625",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 40846,
        "difficulty": null
      },
      {
        "id": "57f3917d19b34b763571ea0b1d6ee8cb85f7d14ae794eb6362527e588ae6d286",
        "domain": "numina",
        "question": "148. A plane intersecting the surface of a triangular pyramid divides the medians of the faces emanating from one vertex in the ratios $2: 1, 1: 2, 4: 1$ respectively (counting from the vertex). In what ratio does this plane divide the volume of the pyramid?",
        "answer": "\\frac{7123}{16901}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29854,
        "difficulty": null
      },
      {
        "id": "2ca6a35f18554f965778f3084bec4ce03079d8e73cf215a75c5903064024693f",
        "domain": "numina",
        "question": "11. Master Wang drives from location A to location B to deliver goods. If he travels at a speed of 60 kilometers per hour both ways, he can return to location A on time. However, upon arriving at location B, he finds that his speed from A to B was only 55 kilometers per hour. If he wants to return to location A on time, at what speed should he drive back?",
        "answer": "66",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146271,
        "difficulty": null
      },
      {
        "id": "ac097b0e195cf09b1f205b11ca25d15b52a4f6670c6a75256c79e4203ad43013",
        "domain": "numina",
        "question": "3. Given $13a - 4b = 11$, then $2013 - (20b - 65a) =$",
        "answer": "2068",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132048,
        "difficulty": null
      },
      {
        "id": "98fee1c15b1dd0b019f62224b00f34051866838febece27cbc55b3138adec0ce",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{2^{n+1}+3^{n+1}}{2^{n}+3^{n}}\n$$",
        "answer": "3",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 45882,
        "difficulty": null
      },
      {
        "id": "7a36f55030007af43d7936c1959ec17edbbd51b6f7b4441d8ff9fae39cd9908d",
        "domain": "numina",
        "question": "Example 3 Given real numbers $x_{1}, x_{2}, \\cdots, x_{10}$ satisfy $\\sum_{i=1}^{10}\\left|x_{i}-1\\right| \\leqslant 4, \\sum_{i=1}^{10}\\left|x_{i}-2\\right| \\leqslant 6$. Find the average value $\\bar{x}$ of $x_{1}, x_{2}, \\cdots, x_{10}$. (2012, Zhejiang Province High School Mathematics Competition)",
        "answer": "1.4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145286,
        "difficulty": null
      },
      {
        "id": "ff4aae135d524fa89cde89e2304adc94f9226a0423c69e3b3f177c3514f4e2ca",
        "domain": "numina",
        "question": "Sure, here is the translation:\n\n---\n\na) What is the value of the following fraction?\n\n$$\n\\frac{423134 \\cdot 846267 - 423133}{423133 \\cdot 846267 + 423134}\n$$\n\nb) What number should be subtracted from the numerator and added to the denominator of the fraction $\\frac{52367}{47633}$ so that after simplification, the value of the fraction is $\\frac{17}{83}$?\n\nc) What is the sum of the following fractions?\n\n$$\n\\frac{7}{2 \\cdot 9} + \\frac{7}{9 \\cdot 16} + \\frac{7}{16 \\cdot 23} + \\cdots + \\frac{7}{65 \\cdot 72}\n$$",
        "answer": "\\frac{35}{72}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 172732,
        "difficulty": null
      },
      {
        "id": "1a85afa5bfeb6ad27ad5981d497010d61e046ffdec9f9c8761dfa9a3ec4263c7",
        "domain": "numina",
        "question": "15. Minimum Sum (9th grade, 4 points). There are a lot of symmetric dice. They are thrown simultaneously. With some probability $p>0$, the sum of the points can be 2022. What is the smallest sum of points that can fall when these dice are thrown with the same probability $p$?",
        "answer": "337",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10720,
        "difficulty": null
      },
      {
        "id": "ed55c1e4b937a04e03d3001920f5ddcdd11dace3e1dc923f44dfbc6a4474ae5a",
        "domain": "numina",
        "question": "7. In the Cartesian coordinate system, it is known that the parabola $y=k x^{2}(k>0)$ intersects the circle $(x-a)^{2}+(y-b)^{2}=r^{2}$ at least at 3 points, one of which is the origin, and the other two lie on the line $y=k x+b$. What is the minimum value of the real number $b$? $\\qquad$ .",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 140916,
        "difficulty": null
      },
      {
        "id": "1c6c0459d16f78e65020e784c9652e055ec23af99843cd2a5db9a59df8377453",
        "domain": "numina",
        "question": "Task 1. On the faces of a six-faced dice, the numbers 1, 2, 3, 4, 5, and 6 are placed. The dice is thrown and falls on the table. After this, the numbers on all faces except one are visible. The numbers on the five visible faces are multiplied. Find the probability that this product is divisible by 16.\n\nAnswer: $0.5$.",
        "answer": "0.5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6564,
        "difficulty": null
      },
      {
        "id": "dbe0bc91d960b662c369ed73252c83b0b36608e5f0b123f449037fd93f3983f1",
        "domain": "numina",
        "question": "Problem 9.4. For positive numbers $a, b, c$, it is known that\n\n$$\n\\frac{a+b+c}{a+b-c}=7, \\quad \\frac{a+b+c}{a+c-b}=1.75\n$$\n\nWhat is the value of $\\frac{a+b+c}{b+c-a}$?",
        "answer": "3.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13056,
        "difficulty": null
      },
      {
        "id": "6dd0f284c2ce1375dae1ae22bac8bf0cdf18411083a3ee2e7385143425ebcda0",
        "domain": "numina",
        "question": "Problem 1. The sum of the first four terms of an arithmetic progression, as well as the sum of the first nine terms, are natural numbers. In addition, its first term $b_{1}$ satisfies the inequality $b_{1} \\leqslant \\frac{3}{4}$. What is the greatest value that $b_{1}$ can take?",
        "answer": "\\frac{11}{15}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9166,
        "difficulty": null
      },
      {
        "id": "d268ef30873cc6f445a96cc69c9ff00aae6b35eae1ffbdacfea0447f3c1912a4",
        "domain": "numina",
        "question": "2. Find the number of natural numbers $k$, not exceeding 454500, such that $k^{2}-k$ is divisible by 505. Answer: 3600.",
        "answer": "3600",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 3033,
        "difficulty": null
      },
      {
        "id": "29bf464a4998ce58fcf9c2041412e7a1b902a0a111a35d6a8236f1d3eb8fe8f3",
        "domain": "numina",
        "question": "Example 17 Real numbers $x_{1}, x_{2}, \\cdots, x_{2001}$ satisfy $\\sum_{k=1}^{2000}\\left|x_{k}-x_{k+1}\\right|=2001$, let $y_{k}=\\frac{1}{k}\\left(x_{1}+x_{2}+\\cdots+\\right.$ $\\left.x_{k}\\right), k=1,2, \\cdots, 2001$. Find the maximum possible value of $\\sum_{k=1}^{2000} \\mid y_{k}-y_{k+1}$ . (2001 Shanghai Competition Problem)",
        "answer": "2000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112951,
        "difficulty": null
      },
      {
        "id": "b1030b1b3ea221f1fb823440270a546b0b27be0e65ff263a04d20713acb2e867",
        "domain": "numina",
        "question": "9.18*. How many sides can a convex polygon have if all its diagonals have the same length?",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 27327,
        "difficulty": null
      },
      {
        "id": "4c7d47716faff076a2f50d271dce0008881630d3c11deaa870e01e7a06a9ef70",
        "domain": "numina",
        "question": "A computer program evaluates expressions without parentheses in the following way:\n\n1) First, it performs the multiplications and divisions from left to right one by one.\n2) Then, it performs the additions and subtractions again from left to right.\n\nFor example, the value of the expression $1-2 / 3-4$ is $-3 \\frac{2}{3}$. How many different results can we get if in the following expression we write one of the signs $+- /, \\times$ independently in each $*$ position?\n\n$1 * 1 * 1 * 1 * 1 * 1 * 1 * 1$",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 168915,
        "difficulty": null
      },
      {
        "id": "b375cd477c4836b949d9590a7401394d6a726e7a3f744ce91e68c0f143df80a0",
        "domain": "numina",
        "question": "Mumchov D:\n\nOn a circle of length 2013, 2013 points are marked, dividing it into equal arcs. A chip is placed at each marked point. We define the distance between two points as the length of the shorter arc between them. For what largest $n$ can the chips be rearranged so that there is again one chip at each marked point, and the distance between any two chips that were initially no more than $n$ apart has increased?",
        "answer": "670",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 61022,
        "difficulty": null
      },
      {
        "id": "457a455b2e6c5e843f703f958194945dcffe995519cd4d1f5ec79ac93c969174",
        "domain": "numina",
        "question": "Problem 11.3. The teacher wrote the number 1818 on the board. Vasya noticed that if a multiplication sign is written between the hundreds and tens digits, the value of the resulting expression is a perfect square $\\left(18 \\times 18=324=18^{2}\\right)$. What is the next four-digit number after 1818 that has the same property?",
        "answer": "1832",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12134,
        "difficulty": null
      },
      {
        "id": "2e0482e2eea388f4ce5fe8e230ce5956e9b7dabf860be9f0471270f82c0a2801",
        "domain": "numina",
        "question": "Example 6. Calculate $\\int_{1}^{e} \\ln x d x$.",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32747,
        "difficulty": null
      },
      {
        "id": "80a08b9543a04cb0f30dadd14825ac9be4c155376b3829e2674de839e020df34",
        "domain": "numina",
        "question": "4. The numbers $x$ and $y$ are such that the equalities $\\operatorname{tg} x - \\operatorname{tg} y = 7$ and $2 \\sin (2 x - 2 y) = \\sin 2 x \\sin 2 y$ hold. Find $\\operatorname{tg} x \\operatorname{tg} y$.",
        "answer": "-\\frac{7}{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3490,
        "difficulty": null
      },
      {
        "id": "fa8f4ce4bc77e1f02d1125c24f72f1755ebd03f85de0353fddc92111a6499ae7",
        "domain": "numina",
        "question": "Given a $3 \\times 3$ table (like for playing tic-tac-toe). Four chips are placed randomly in four randomly chosen cells.\n\nFind the probability that among these four chips, there will be three that are in a row either vertically, horizontally, or diagonally.",
        "answer": "\\frac{8}{21}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55851,
        "difficulty": null
      },
      {
        "id": "2cfe9f2f3d2e499fb035a046e198ace71fb9cd66f945b67480f90422c178048c",
        "domain": "numina",
        "question": "20. Piplu is learning long division, and he finds that when a three-digit number is divided by 19, the quotient is $a$, and the remainder is $b$ ($a, b$ are natural numbers), then the maximum value of $a+b$ is . $\\qquad$",
        "answer": "69",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 136423,
        "difficulty": null
      },
      {
        "id": "36ae001f5d7eeb80999e18de4d43fcb587fd2b1914b39aee994bf0ea0cf2038c",
        "domain": "numina",
        "question": "5. Let $f(x)=x^{3}+3\\left(x^{2}+x+\\sin \\pi x\\right)$, then $f(1-\\pi)+f(\\pi-3)=$",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 123912,
        "difficulty": null
      },
      {
        "id": "3b6c869a587999b01554629472b9d4096df0d007f0d4f0182d95c1bc735179aa",
        "domain": "numina",
        "question": "Example 4-11 The side chain of methane $\\mathrm{CH}_{4}$ is , if the 4 H bonds are replaced by one of $\\mathrm{H}, \\mathrm{Cl}, \\mathrm{CH}_{3}, \\mathrm{C}_{2} \\mathrm{H}_{5}$, how many different chemical structures are there?",
        "answer": "36",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 103410,
        "difficulty": null
      },
      {
        "id": "fbd6804110ff05d93708080d531e7d16d1dd1253f7654f07c33c9feb096308c8",
        "domain": "numina",
        "question": "18. Traffic Light (from 10th grade. 2 points). A traffic light at a pedestrian crossing allows pedestrians to cross the street for one minute and prohibits crossing for two minutes. Find the average waiting time for the green light for a pedestrian who approaches the intersection.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2e6196530ac8ddeae41bg-13.jpg?height=314&width=331&top_left_y=1308&top_left_x=291)",
        "answer": "40",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 10977,
        "difficulty": null
      },
      {
        "id": "a70d37c61fc1b2be9047d89b01bf1aa44ab65199404991a999227d7dffbc110b",
        "domain": "numina",
        "question": "Péter took one of each of the stamps worth $1, 2, 3, \\ldots, 37$ forints from his stamp collection. He would like to group these stamps so that the sum of the face values of the stamps in each group is the same. In how many different ways can he do this?",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 171132,
        "difficulty": null
      },
      {
        "id": "157d6e27b9ae50d39d199af5e0e269d686104b6ba07f9204f743ad8186aeaf3c",
        "domain": "numina",
        "question": "## Task 1 - 070611\n\nA certain large city's tram station has a total of 83 trams. Of these, exactly 46 are trailers. At a certain point in time, a total of 8 motor cars, each with two trailers, and 23 motor cars, each with one trailer, are in service.\n\nWhat is the number of all motor cars and trailers that are not in service at this time?",
        "answer": "7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 183219,
        "difficulty": null
      },
      {
        "id": "d06d8a7a510bbf364b4d4075a289cec65af41a35f5ea45a7f852778a11d2ac45",
        "domain": "numina",
        "question": "B3 Alpine Expedition has 420 kg of luggage.\n\nA What is the minimum number of porters they need to hire if each porter can carry a maximum of 35 kg of load?\n\nB On average, how much weight should each porter carry if there are 15 of them?\n\nC In how many ways can the first three porters choose their loads if they are choosing from ten different loads?\n\nD One of the loads is in the shape of a cuboid, and the other is in the shape of a cylinder with a diameter of $1 \\mathrm{~m}$ and a height of $0.8 \\mathrm{~m}$. The cuboid is $1.57 \\mathrm{~m}$ long and $4 \\mathrm{dm}$ wide. How high is it if both loads have the same volume?",
        "answer": "1",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 163362,
        "difficulty": null
      },
      {
        "id": "652f7c78bd0b4cf3300234117c8a9b36b9f494b58651e953bf364040c2eaa24e",
        "domain": "numina",
        "question": "7. Consider the set $X=\\{1,2,3, \\ldots, 9,10\\}$. Find two disjoint nonempty subsets $A$ and $B$ of $X$ such that\n\n(a) $A \\cup B=X$;\n\n(b) $\\operatorname{prod}(A)$ is divisible by $\\operatorname{prod}(B)$, where for any finite set of numbers $C, \\operatorname{prod}(C)$ denotes the product of all numbers in $C$;\n\n(c) the quotient $\\operatorname{prod}(A) / \\operatorname{prod}(B)$ is as small as possible.",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 180491,
        "difficulty": null
      },
      {
        "id": "6c9a3ee448947e1e4ac39adc867d59214b3e4fc451798d8eda91b87940fddd2d",
        "domain": "numina",
        "question": "5. As shown in the figure, a large triangle $A B C$ is divided into seven parts by three line segments, four of which are triangles and the other three are quadrilaterals. The sum of the perimeters of the three quadrilaterals is 25 cm, the sum of the perimeters of the four triangles is 20 cm, and the perimeter of triangle $A B C$ is 19 cm. Then $A D+B E+C F=$ $\\qquad$ cm",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 131926,
        "difficulty": null
      },
      {
        "id": "6be4a8501cf59d7a04fe81d6660597dc79c4f8a9804fc5745191a7edb24bf5dd",
        "domain": "numina",
        "question": "6.3. At a round table sit 10 elves, each with a basket of nuts in front of them. Each was asked, \"How many nuts do your two neighbors have together?\" and, going around the circle, the answers received were 110, 120, 130, 140, 150, 160, 170, 180, 190, and 200. How many nuts does the elf who answered 160 have?",
        "answer": "55",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 4438,
        "difficulty": null
      },
      {
        "id": "43c5b0a88c7bf85baec062556ee3452c7aa8c9f5a2f1aa5aa4596eed87f09a5d",
        "domain": "numina",
        "question": "1. Given a linear function $f(x)$. It is known that the distance between the points of intersection of the graphs $y=$ $x^{2}-1$ and $y=f(x)$ is $\\sqrt{30}$, and the distance between the points of intersection of the graphs $y=x^{2}$ and $y=f(x)+3$ is $\\sqrt{46}$. Find the distance between the points of intersection of the graphs of the functions $y=x^{2}-1$ and $y=f(x)+1$.",
        "answer": "\\sqrt{38}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3082,
        "difficulty": null
      },
      {
        "id": "0e00639da94f02bf8fa97bd19d2e2352ca03ac0eb38be5aa9483c04d45d01359",
        "domain": "numina",
        "question": "12. Let real numbers $x, y, z$ satisfy $xy \\neq 0, x+y \\neq 0$. Find\n$$\n\\left(\\frac{y+z}{x}+2\\right)^{2}+\\left(\\frac{z}{y}+2\\right)^{2}+\\left(\\frac{z}{x+y}-1\\right)^{2}\n$$\n\nthe minimum value.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142909,
        "difficulty": null
      },
      {
        "id": "3d8d62fa9ed7d7360efa3f33d4aabdb94c9aa665be6c5346055c83c5151917fe",
        "domain": "numina",
        "question": "8. Let $x, y$ satisfy the constraint conditions $\\left\\{\\begin{array}{l}x-y-2 \\geqslant 0, \\\\ x-5 \\leqslant 0, \\\\ y+2 \\geqslant 0 .\\end{array}\\right.$ Then the sum of the minimum and maximum values of $z=x^{2}+y^{2}$ is $\\qquad$",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112689,
        "difficulty": null
      },
      {
        "id": "c078d5801086d37c7f97e8878bdbd59c711412768dadcd2b4f037b3f0c692972",
        "domain": "numina",
        "question": "3. (17 points) In triangle $A B C \\quad A B=4, B C=6$, angle $A B C$ is $30^{\\circ}, B D-$ is the bisector of triangle $A B C$. Find the area of triangle $A B D$.",
        "answer": "2.4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 4026,
        "difficulty": null
      },
      {
        "id": "070ef55aa4f915d658099143714190ae7b4e3c2041f9d4429d4e16eaa1a83f0a",
        "domain": "numina",
        "question": "1. Buses from Moscow to Oryol depart at the beginning of each hour (at 00 minutes). Buses from Oryol to Moscow depart in the middle of each hour (at 30 minutes). The journey between the cities takes 5 hours. How many buses from Oryol will the bus that left from Moscow meet on its way?",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 4575,
        "difficulty": null
      },
      {
        "id": "7fa8664708d5fd285f479f5f00a2bf8998682e774e779f8f2178c51163417f92",
        "domain": "numina",
        "question": "29.1.3 Lin has a $9 \\times 2004$ grid, in which the positive integers from 1 to 2004 are each filled in 9 times. And in each column, the sum of the numbers filled in is no more than 3. Find the minimum possible value of the sum of the numbers in the first row.",
        "answer": "2005004",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 142200,
        "difficulty": null
      },
      {
        "id": "e8fac569e326f3d576162d9a7f8997fa3a19b1b3aa2aea2d23b711cd9cbda007",
        "domain": "numina",
        "question": "Maruška wrote two different natural numbers on the board. Marta took a card and on one side of it, she wrote the sum of Maruška's numbers, and on the other side, their product. One of the numbers on the card was a prime number, and the sum of the numbers on both sides of the card was 97.\n\nWhich numbers did Maruška write on the board?\n\n(L. Hozová)",
        "answer": "613",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76246,
        "difficulty": null
      },
      {
        "id": "7dadfd6fbd05297d8cecd983b4f370084a7f0888412600b8a96e81abd28a6b1d",
        "domain": "numina",
        "question": "Question 114, Given the equation $\\mathrm{x}^{10}+(13 \\mathrm{x}-1)^{10}=0$ has 5 pairs of conjugate complex roots $\\mathrm{r}_{\\mathrm{k}} 、 \\overline{\\mathrm{r}_{\\mathrm{k}}}(1 \\leq \\mathrm{k} \\leq 5)$, try to find the value of $\\sum_{\\mathrm{k}=1}^{5} \\frac{1}{\\mathrm{r}_{\\mathrm{k}} \\cdot \\mathrm{r}_{\\mathrm{k}}}$.",
        "answer": "850",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143813,
        "difficulty": null
      },
      {
        "id": "ada06be4f1a6fe26fa52b752f8e6de7e664df14bb0720d93c32d61c5156c1c9a",
        "domain": "numina",
        "question": "What is the last digit of $2019^{2020^{2021}}$?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 80455,
        "difficulty": null
      },
      {
        "id": "e453a6f5dab8d4dd21aa37c595217638c62fa22b2a736963d8c311725c67e566",
        "domain": "numina",
        "question": "4. Find four real numbers $x_{1}, x_{2}, x_{3}, x_{4}$, such that each, when added to the product of the others, equals two.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4642,
        "difficulty": null
      },
      {
        "id": "8f6da5618433826e820af784ff1bd54d8076d39e3f258f4fde232d3f88ff4f98",
        "domain": "numina",
        "question": "20. Given that $[x]$ represents the greatest integer not exceeding $x$, such as $[1.5]=1,[2]=2$, then the last two digits of $\\left[\\frac{10^{2040}}{10^{60}-3}\\right]$ are . $\\qquad$",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 135641,
        "difficulty": null
      },
      {
        "id": "05d59e0ab49229a0acde3f1d111dc85beadbfa259bdf855880c6ff9124a7ab4c",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nIf $2 t+2 x-t-3 x+4 x+2 t=30$, what is the value of $x$ ?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194163,
        "difficulty": null
      },
      {
        "id": "c061e3ea7d3e1e36f88f9d16e349bdd0e131b928b29a0239679c17606cce1102",
        "domain": "numina",
        "question": "\\section*{Problem 6 - 021016}\n\nDetermine the smallest natural number \\(n\\) that has the following properties:\n\na) its decimal representation ends with the digit 6;\n\nb) if this last digit 6 is removed and written as the first digit before the other unchanged digits, the result is four times the number \\(n\\).",
        "answer": "153846",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 184872,
        "difficulty": null
      },
      {
        "id": "662bf7d733bc1633ecd85b2d4bb314f4f6e546eef3f8f34bb4a4695d52967487",
        "domain": "numina",
        "question": "8. A coin collector has 100 coins that look the same. The collector knows that 30 of them are genuine, 70 are fake, and that all genuine coins weigh the same, while all fake coins weigh different and are heavier than the genuine ones. The collector has a balance scale that can be used to compare the weight of two groups of coins of the same number. To ensure finding at least one genuine coin, how many weighings are needed at minimum?",
        "answer": "70",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 117604,
        "difficulty": null
      },
      {
        "id": "2f567d83422ca2bb81903b78b2de9ad674b60888d985b609bf6d27bf1e66c16c",
        "domain": "numina",
        "question": "85. In a triangle, two sides are given. At what angle between them will the area of the triangle be the greatest?",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 44665,
        "difficulty": null
      },
      {
        "id": "50e2a912c6fdb47eb62bef6365a587f068367ef9e3d1f9c4f2b54b4f28dd2d71",
        "domain": "numina",
        "question": "11.1. When composing the options for the district mathematics olympiad for grades $7, 8, 9, 10, 11$, the jury aims to ensure that in the option for each grade, there are exactly 7 problems, of which exactly 4 do not appear in any other option. What is the maximum number of problems that can be included in the olympiad?",
        "answer": "27",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17972,
        "difficulty": null
      },
      {
        "id": "a491247eec8736e3e22b3d910f38b169aecddf4e8484e21f495454c22af75710",
        "domain": "numina",
        "question": "5-1. A square with a side of 100 was cut into two equal rectangles. They were placed next to each other as shown in the figure. Find the perimeter of the resulting figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d2d35e627535cd91d6ebg-04.jpg?height=277&width=594&top_left_y=684&top_left_x=731)",
        "answer": "500",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 16451,
        "difficulty": null
      },
      {
        "id": "eb2d12027b30b85d05e99aa339d432aaf79951df084657a966eaf6895710b3b9",
        "domain": "numina",
        "question": "3. On three islands, there are 2019 cities, and some two-way air routes have been opened between them. Each route connects three cities on different islands. Any two cities are on at most one route. Find the maximum number of routes.",
        "answer": "673^2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 141023,
        "difficulty": null
      },
      {
        "id": "31c3e9eb6d3d7fa08494b91cf135ff821912e8655ad3ebab3ff5443500865104",
        "domain": "numina",
        "question": "44. Given 5 points, no three of which lie on the same line. Find the probability that, by randomly selecting two points, a student will get the required line.",
        "answer": "\\frac{1}{10}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 32269,
        "difficulty": null
      },
      {
        "id": "40cc8690488c5ae9188aaf4aae32a44d8a0bc14f8bd7e2cbd7f23ba62ffc9541",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=2 p-q$\n\n$b=p+3 q$\n\n$|p|=3$\n\n$|q|=2$\n\n$(\\widehat{p, q})=\\frac{\\pi}{2}$",
        "answer": "42",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 47398,
        "difficulty": null
      },
      {
        "id": "8175ed6de0830a682cc0b3821eb5db7624474b63cbcf8ee2458309b672870b26",
        "domain": "numina",
        "question": "Let's calculate the following sum:\n\n$$\n\\sqrt{7-\\sqrt{48}}+\\sqrt{5-\\sqrt{24}}+\\sqrt{3-\\sqrt{8}}\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 164369,
        "difficulty": null
      },
      {
        "id": "d271c6c08e2b6f169a2f7d26af2140b88c489c9454b0b5b49c5d11cbd05c2180",
        "domain": "numina",
        "question": "1. Distribute 5 different books to 3 students, with each student getting at least one book. How many different ways are there to do this?",
        "answer": "150",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 141496,
        "difficulty": null
      },
      {
        "id": "547b352b3cf45cd1ff0f60f86681c0df4f4698935a788b5c25f89fdcfe76aafd",
        "domain": "numina",
        "question": "Vasilev N.B.\n\nA cherry, a sphere of radius $r$, is dropped into a round glass, the axial cross-section of which is the graph of the function $y=x^{4}$. For what largest $r$ will the sphere touch the lowest point of the bottom? (In other words, what is the maximum radius $r$ of a circle lying in the region $y \\geq x^{4}$ and containing the origin?)",
        "answer": "\\frac{3\\sqrt[3]{2}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 70546,
        "difficulty": null
      },
      {
        "id": "5eb09b420b42e30484486aa50e600c7466387431b8eab2d100b526c4942ea3e7",
        "domain": "numina",
        "question": "11. On an elliptical track with a perimeter of 400 meters, two people, A and B, start cycling from two points 300 meters apart and head towards each other along the track. After meeting, they continue to move forward. It is known that A's cycling speed is 4 meters/second, and B's cycling speed is 6 meters/second. When they meet for the 6th time, the two have cycled for at least (\n) seconds",
        "answer": "210",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98738,
        "difficulty": null
      },
      {
        "id": "525745091638406228064272f5154022eba5c2912991b2fe1bff603fd3d6b729",
        "domain": "numina",
        "question": "A store was selling each unit of a toy for $R \\$ 13.00$. To sell its entire stock, which was not more than 100 units, the store management decided to lower the price by an integer number of reais, thus obtaining $\\mathrm{R} \\$ 781.00$ for the entire stock. What was the price reduction, per unit?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74413,
        "difficulty": null
      },
      {
        "id": "e1cfa45bba4964f7c84e0c3a4e77764b5194050d85d494274477d7f7eb1ac39e",
        "domain": "numina",
        "question": "11. (20 points) Let point $Z$ move on the circle $|z|=3$ in the complex plane, and let $w=\\frac{1}{2}\\left(z+\\frac{1}{z}\\right)$. The trajectory of $w$ is the curve $\\Gamma$. A line $l$ passes through the point $P(1,0)$ and intersects the curve $\\Gamma$ at points $A$ and $B$, and intersects the imaginary axis at point $M$. If $\\overrightarrow{M A}=t \\overrightarrow{A P}$ and $\\overrightarrow{M B}=s \\overrightarrow{B P}$, find the value of $t+s$.",
        "answer": "-\\frac{25}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109172,
        "difficulty": null
      },
      {
        "id": "b0225e58ba37b80a96630511a4fbb4050b56e420ee78f66ddd2673d6e5bad8e1",
        "domain": "numina",
        "question": "1.1. Arina wrote down a number and the number of the month of her birthday, multiplied them and got 248. In which month was Arina born? Write the number of the month in your answer.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6c94238ddc8917ebef5dg-1.jpg?height=142&width=874&top_left_y=266&top_left_x=585)",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 13742,
        "difficulty": null
      },
      {
        "id": "dd69d24413e5d8a102c82790cdd7e90cf434ff00be0150d1138f714b7ea2f0d0",
        "domain": "numina",
        "question": "5. “Xiong Da” $\\times$ “Xiong Er” = “Xiong Xiong Di” . If the same Chinese character represents the same digit from 0 to 9, different Chinese characters represent different digits, and “Da” > “Er” , then the sum of all three-digit numbers represented by “Xiong Xiong Di” that satisfy the conditions is $\\qquad$ . “Da”+“Er” $<9$;",
        "answer": "686",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 103377,
        "difficulty": null
      },
      {
        "id": "f8280d0c8631cdb5a6b8ad6792d8dd14431c74aaef601594d29f625a922bcd70",
        "domain": "numina",
        "question": "14. From the 20 numbers 11, 12, 13, 14, ... 30, at least ( ) numbers must be taken to ensure that there are two numbers whose sum is a multiple of ten.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 111478,
        "difficulty": null
      },
      {
        "id": "e5f7bf307737b2fbbebe2c3f4096f49de007fe085a47b2067a5b39e3b57f37f6",
        "domain": "numina",
        "question": "1. Find all positive integers $n$, such that $3^{n}+n^{2}+2019$ is a perfect square.\n（Zou Jin, contributed）",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124674,
        "difficulty": null
      },
      {
        "id": "79eae271f1d434736a1c1110b7a57caa83f7ffb5f725e274ae13851698445935",
        "domain": "numina",
        "question": "1. For some real number $c$, the graphs of the equation $y=|x-20|+|x+18|$ and the line $y=x+c$ intersect at exactly one point. What is $c$ ?",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151679,
        "difficulty": null
      },
      {
        "id": "6cb7c50c9927925c4a0e2056c61e3b16c5cdc04af3c6c36af5f74c9d5a2fbcd5",
        "domain": "numina",
        "question": "In triangle $ABC$, the height $BD$ is 6, the median $CE$ is 5, and the distance from point $K$ of intersection of segments $BD$ and $CE$ to side $AC$ is 1.\n\nFind the side $AB$.",
        "answer": "\\frac{2\\sqrt{145}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67209,
        "difficulty": null
      },
      {
        "id": "ef25cb055bc5788c59a5d2e1baf38da6f968b10180fa6714b838d129eb98b4de",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nEach of the three lines having equations $x+t y+8=0,5 x-t y+4=0$, and $3 x-k y+1=0$ passes through the same point. What is the value of $k$ ?\n\n##",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194119,
        "difficulty": null
      },
      {
        "id": "33a02b2feb00d1f1f2ae5c9de2a8a5885bbcb1993d130d2b4c310e697244d707",
        "domain": "numina",
        "question": "[ [Assist\n\nIn triangle $ABC$, the height $BD$ is 11.2 and the height $AE$ is 12. Point $E$ lies on side $BC$ and $BE$ : $EC=5: 9$. Find the side $AC$.\n\n#",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58762,
        "difficulty": null
      },
      {
        "id": "16979cc7fffe012a738a77e6a4a9386876e7b99edacd0a2128422e2e1dc01747",
        "domain": "numina",
        "question": "1. Calculate: $1325: 5+15: 5-(99-9: 9) \\cdot 2-35-30$.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86688,
        "difficulty": null
      },
      {
        "id": "39dfaf072f3d40041afa6e1b8878652010a22d2808ad05298d76b540a5e894c8",
        "domain": "numina",
        "question": "1. Determine the smallest natural number that is divisible by 63 and whose sum of digits is equal to 63.",
        "answer": "19899999",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188392,
        "difficulty": null
      },
      {
        "id": "20ef1986c0a5bc9d7209c080c1cc7b56e6d04c09f7f93c9a3a1cbeaae57fc696",
        "domain": "numina",
        "question": "Task 4.1. Assign each letter a digit $1,2,3,4,5$ so that all inequalities are satisfied.\n\n$$\n3>\\mathrm{A}>\\mathrm{M}<\\mathrm{E}<\\mathrm{H}<\\mathrm{A}\n$$\n\nDifferent letters must correspond to different digits. Write the number ZAMENA as your answer.",
        "answer": "541234",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13978,
        "difficulty": null
      },
      {
        "id": "066ced87863332305f4fa5ba71936b75e7eb2d98d78d31d5936db737b4a1448b",
        "domain": "numina",
        "question": "1. a) Verify the identity $\\sqrt{1+\\frac{1}{n^{2}}+\\frac{1}{(n+1)^{2}}}=1+\\frac{1}{n}-\\frac{1}{n+1}, \\forall n \\in \\mathbb{N}^{*}$.\n\nb) Determine the integer part of the number\n\n$$\na=\\sqrt{1+\\frac{1}{1^{2}}+\\frac{1}{2^{2}}}+\\sqrt{1+\\frac{1}{2^{2}}+\\frac{1}{3^{2}}}+\\ldots+\\sqrt{1+\\frac{1}{2012^{2}}+\\frac{1}{2013^{2}}}\n$$",
        "answer": "2012",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178364,
        "difficulty": null
      },
      {
        "id": "21d069bd00f6651456153e68b10bb21947eec7d043a64bc5f14b4882955dbed7",
        "domain": "numina",
        "question": "Draw an acute isosceles triangle inside a circle. Over one of the sides of this triangle, construct another isosceles triangle such that its apex also lies on the circumference of the circle. Continuing this process, determine the limit of the angles at the vertices of the triangles.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 176765,
        "difficulty": null
      },
      {
        "id": "356629544c4772701f174ec3fd22205be355fcf0f9d350425462f17be98bcf85",
        "domain": "numina",
        "question": "5. Segment $A B=8$ intersects plane $\\alpha$ at an angle of $30^{\\circ}$ and is divided by this plane in the ratio $1: 3$. Find the radius of the sphere passing through points $A$ and $B$ and intersecting plane $\\alpha$ in a circle of the smallest radius.",
        "answer": "2\\sqrt{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6373,
        "difficulty": null
      },
      {
        "id": "aa862211df44d58118ef713000aee54ff9e18da85c51d55028705a60c9863583",
        "domain": "numina",
        "question": "Variant 7.8.2. Real numbers $a, b, c$ are such that\n\n$$\n\\left\\{\\begin{array}{l}\n\\frac{a b}{a+b}=3 \\\\\n\\frac{b c}{b+c}=5 \\\\\n\\frac{c a}{c+a}=8\n\\end{array}\\right.\n$$\n\nFind the value of the expression $\\frac{a b c}{a b+b c+c a}$.",
        "answer": "\\frac{240}{79}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16029,
        "difficulty": null
      },
      {
        "id": "15720c2a75c13460090eb96de2a35f4fcc61ce7c3ba08db4438eaf40ed8da20d",
        "domain": "numina",
        "question": "5. On a long stick, there are three types of graduation lines. The first type divides the stick into 10 equal parts;\n\nThe second type divides the stick into 12 equal parts; The third type divides the stick into 15 equal parts. If each graduation line cuts the stick, how many pieces will the stick be cut into?",
        "answer": "28",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 111323,
        "difficulty": null
      },
      {
        "id": "e5b4aad2ca7da3316e10620e0b2f1e145e7219396892c846efb08591e2e39599",
        "domain": "numina",
        "question": "9. The function $f(x)=$ $\\frac{x+1}{x}+\\frac{x+2}{x+1}+\\cdots+\\frac{x+2023}{x+2022}$ has a center of symmetry at $(a, b)$, then $2 a+b=$ $\\qquad$ _.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104871,
        "difficulty": null
      },
      {
        "id": "12dfc67517c35a263bea0e3eaedee08009aaa8de5f6f5d1c7268b6d1d3a5c2ad",
        "domain": "numina",
        "question": "42nd Putnam 1981 Problem A1 Let the largest power of 5 dividing 1 1 2 2 3 3 ... n n be 5 f(n) . What is lim n→∞ f(n)/n 2 ? Solution",
        "answer": "\\frac{1}{8}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 148368,
        "difficulty": null
      },
      {
        "id": "f07c6510378a7a245f1df307bc3e4ec7781e13f8de42376edbf506c1a46eb698",
        "domain": "numina",
        "question": "14. Find the value of $\\sin \\frac{\\pi}{7} \\sin \\frac{2 \\pi}{7} \\sin \\frac{3 \\pi}{7}$.",
        "answer": "\\frac{\\sqrt{7}}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128947,
        "difficulty": null
      },
      {
        "id": "f6c11f26f65516ac461d1e996bf057930d5ba1a764611f6e69e7f27d67b263e1",
        "domain": "numina",
        "question": "KöMaL Survey on a total of 14 questions, with 4 possible answers for each question. The best result was 10 correct answers. What is the probability that we achieve this result by random filling out?",
        "answer": "0.00030205",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 170856,
        "difficulty": null
      },
      {
        "id": "c57428f300bb7c0c77444e268362f34f5bade54058c87b2835bd411a1a9cae35",
        "domain": "numina",
        "question": "6. Let's call the distance between numbers the absolute value of their difference. It is known that the sum of the distances from twenty-five consecutive natural numbers to some number $a$ is 1270, and the sum of the distances from these same twenty-five numbers to the number $a^2$ is 1234. Find all possible values of $a$.",
        "answer": "-\\frac{4}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3534,
        "difficulty": null
      },
      {
        "id": "35c04505d11e3f3ff23c7b13cd9678250be2c1b5fe0bd98c3d420d1ec6192c14",
        "domain": "numina",
        "question": "On a $2014 \\times 2014$ board, each of the $2014^{2}$ squares contains a lamp. Lamps can be on or off. In the initial situation, some of the lamps are on. In a move, you choose a row or column in which at least 1007 lamps are on and change the status of all 2014 lamps in that row or column (from on to off and from off to on). Find the smallest non-negative integer $k$ such that the following holds: from any initial situation, you can reach a situation in a finite number of steps where at most $k$ lamps are on.",
        "answer": "1006 \\cdot 2014",
        "topic": "Combinatorics",
        "source": "olympiads_ref",
        "source_index": 195293,
        "difficulty": null
      },
      {
        "id": "1ac70a5484cef8d7d22b2db0806a888bb91beb96acf69c26f00710b3d9037b96",
        "domain": "numina",
        "question": "248. Cats and Mice. One morning at Professor Rackbrain's table, the lively discussion was about the extermination of rodents when suddenly the professor said:\n\n- If a certain number of cats ate a total of 999,919 mice, and all the cats ate the same number of mice, how many cats were there in total?\n\nSomeone suggested that perhaps one cat ate all the mice, but Rackbrain objected, saying he had said \"cats.\" Then someone else ventured to suggest that each cat had eaten one mouse, to which the professor remarked that he had said \"mice.\" He also added, to help those present, that each cat had eaten more mice than there were cats.\n\nWhat, then, would be the correct answer?",
        "answer": "991",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41164,
        "difficulty": null
      },
      {
        "id": "acd936f65395bbf2303968de8c61cf869abda3ec302799ddf871909c1ef187af",
        "domain": "numina",
        "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_038330058549e9b1ba1cg-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
        "answer": "43",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14281,
        "difficulty": null
      },
      {
        "id": "302d91d906bca48899d10b9606c3faea64364b1858408962ec4286ea490f938f",
        "domain": "numina",
        "question": "The ninety-sixth question: Given $S=\\{(\\mathrm{i}, \\mathrm{j}) \\mid \\mathrm{i}, \\mathrm{j}=1, 2, \\ldots, 100\\}$ as the set of $100 \\times 100$ integer points on the Cartesian plane. Each point in $S$ is colored with one of four given colors. Find the maximum possible number of rectangles with vertices of four different colors and sides parallel to the coordinate axes.",
        "answer": "9375000",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 135050,
        "difficulty": null
      },
      {
        "id": "fcc0166fdf29302f320414deebe3bba45955a2409e8cfd7644e729918a8188e0",
        "domain": "numina",
        "question": "$7 \\cdot 67$ Given 11 sets $M_{1}, M_{2}, \\cdots, M_{11}$, where each set has exactly 5 elements and for all $i, j, 1 \\leqslant i<j \\leqslant 11$, there is $M_{i} \\cap M_{j} \\neq \\varnothing$, find the minimum possible value of the maximum number of sets among these sets whose intersection is non-empty.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 118473,
        "difficulty": null
      },
      {
        "id": "f0016ccf57e930f43c52630450fd409e14bfa5a9609d694a79830695e2c4057f",
        "domain": "numina",
        "question": "How many digits are needed to write down the numbers from 1 to 6786?",
        "answer": "26037",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 174323,
        "difficulty": null
      },
      {
        "id": "b32599af522291e1392ec2940b383cd1ad6b11548c222e4b08ecbf4374204958",
        "domain": "numina",
        "question": "8. Aladdin found 20 necklaces in the treasure room, some of which are set with 2 diamonds, and some are set with 5 diamonds, with a total of 79 diamonds. So, the number of necklaces set with 5 diamonds is $\\qquad$ pieces",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93192,
        "difficulty": null
      },
      {
        "id": "ca732749a0696a70cfaff0243516e2aad801ee021d6f8d7c05903305ead975f2",
        "domain": "numina",
        "question": "1. [5] A positive integer is called primer if it has a prime number of distinct prime factors. Find the smallest primer number.",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151764,
        "difficulty": null
      },
      {
        "id": "b771ccf75961a16d7c6f64b3491ffab9a39e9120d664302a214cadc1e07a1fec",
        "domain": "numina",
        "question": "8.5. Kolya drew 10 segments and marked all their intersection points in red. Counting the red points, he noticed the following property: on each segment, there are three red points. a) Provide an example of the arrangement of 10 segments with this property. b) What is the maximum number of red points for 10 segments with this property?",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 16301,
        "difficulty": null
      },
      {
        "id": "67d241649c97cf73c2629ae2a228dc99f21c674e0964c0ac20a39dbc04667394",
        "domain": "numina",
        "question": "11. (12 points) Today is December 19, we place the numbers $1 、 2 、 1 、 9$ composed of small shaded squares with a side length of 1 in an $8 \\times 5$ large rectangle. When the large rectangle is rotated $180^{\\circ}$, it becomes “6121”. If these two $8 \\times$ 5 large rectangles are overlapped, then the number of overlapping shaded squares is $\\qquad$.",
        "answer": "30",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 141941,
        "difficulty": null
      },
      {
        "id": "0ce1331f2479c7985c282dfa457dd9c726fe63332126a0b8e1d78ac0d9ac8a49",
        "domain": "numina",
        "question": "2. At a cactus lovers' meeting, 80 cactus enthusiasts presented their collections, each consisting of cacti of different species. It turned out that no species of cactus is present in all collections at once, but any 15 people have cacti of the same species. What is the smallest total number of cactus species that can be in all the collections?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 8049,
        "difficulty": null
      },
      {
        "id": "43942cd95fa41abd9f42a247ffcfe4f86c867ed7a11a0ebef0d69edb632a39cf",
        "domain": "numina",
        "question": "[Ratio of areas of similar triangles]\n\nFind the area of a triangle whose medians are 12, 15, and 21.\n\n#",
        "answer": "48\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 60160,
        "difficulty": null
      },
      {
        "id": "f1dca08c9581c7cf909969249ed97ce46cb8bead7217b38268f95004b2763c96",
        "domain": "numina",
        "question": "[ Coordinate method on the plane [ The ratio of the areas of triangles with a common base or common height]\n\n ] On the coordinate plane, points $A(1 ; 3), B(1 ; 9), C(6 ; 8)$, and $E(5 ; 1)$ are given. Find the area of the pentagon $A B C D E$, where $D$ is the point of intersection of the lines $A C$ and $B E$.",
        "answer": "21",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 52549,
        "difficulty": null
      },
      {
        "id": "05268a8032def6fefcdc0a7650119033eeb48082ebd3eac8371b313150521c8f",
        "domain": "numina",
        "question": "3B. Let $a, b, c$ be the sides of a triangle. Determine the angle opposite side $c$, if\n\n$$\n(a+b+c)(a+b-c)=3 a b\n$$",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 190359,
        "difficulty": null
      },
      {
        "id": "9cecfb27a6f6ee51c04cb84def8b717c552cfd93601c8e42fad4ab3893f964f5",
        "domain": "numina",
        "question": "3. [5] How many six-digit multiples of 27 have only 3,6 , or 9 as their digits?",
        "answer": "51",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151858,
        "difficulty": null
      },
      {
        "id": "b0457f25d941aa52b29f23cd0880e23d045fad5722c3f41f1cf1c3da9c75382a",
        "domain": "numina",
        "question": "5. Given four points $A, B, C, D$ on a unit sphere, satisfying $AB=AC=AD, BC=CD=DB$. Then the maximum volume of tetrahedron $ABCD$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{8\\sqrt{3}}{27}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 128831,
        "difficulty": null
      },
      {
        "id": "830c88dce2193c5acc00734438dcd7efa50231ac95540931d18484f3d2fbd8da",
        "domain": "numina",
        "question": "6. Find the smallest natural number $n$ such that for any coloring of the numbers 1, 2, 3, ..., $n$ with three colors, there exist two numbers of the same color whose difference is a square of a natural number.\n\n(Vojtech Bálint, Michal Rolínek, Josef Tkadlec)",
        "answer": "29",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 75218,
        "difficulty": null
      },
      {
        "id": "bc4067ded4cf80945c241ba9b8d369a1424280503900d383d6551fcefeab0135",
        "domain": "numina",
        "question": "1. A robot is located in one of the cells of an infinite grid and can be given the following commands:\n\n- up (the robot moves to the adjacent cell above);\n- down (the robot moves to the adjacent cell below);\n- left (the robot moves to the adjacent cell to the left);\n- right (the robot moves to the adjacent cell to the right).\n\nFor example, if the robot executes a sequence of four commands (up, right, down, left), it will obviously return to its initial position, i.e., it will end up in the same cell from which it started. How many different sequences of 4 commands exist that return the robot to its initial position?",
        "answer": "36",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 8709,
        "difficulty": null
      },
      {
        "id": "9ba50ae6d9e50bb3394594ba49f9f11a4a58546d321e384604e63a7964be620c",
        "domain": "numina",
        "question": "Problem 9.5. Point $M$ is the midpoint of side $B C$ of triangle $A B C$, where $A B=17$, $A C=30, B C=19$. A circle is constructed with side $A B$ as its diameter. An arbitrary point $X$ is chosen on this circle. What is the minimum value that the length of segment $M X$ can take?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2befe970655743580344g-03.jpg?height=414&width=553&top_left_y=1517&top_left_x=448)",
        "answer": "6.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13057,
        "difficulty": null
      },
      {
        "id": "0bdeb89f13087e99b85d68ee6e675ee7db372920012dbd7f97c4038418f4a340",
        "domain": "numina",
        "question": "Problem 3. Given an acute triangle $ABC$. Its altitudes $AE$ and $CD$ intersect at point $H$, and $\\overline{AB} = \\overline{CH}$. Calculate the angle $\\measuredangle ACB$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188510,
        "difficulty": null
      },
      {
        "id": "23810ed7bde60466aceef684f4fa18826851a2efee3c9962f1f21f849eee233e",
        "domain": "numina",
        "question": "3. Let $A B C$ be a triangle with $\\angle B A C=90^{\\circ}$. Points $D, E$ on its hypotenuse $B C$ satisfy $C D=C A, B E=B A$. Let $F$ be such a point inside $\\triangle A B C$ that $D E F$ is an isosceles right triangle with hypotenuse DE. Find $\\angle B F C$. (Patrik Bak)",
        "answer": "135",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 77000,
        "difficulty": null
      },
      {
        "id": "7d968e6e043958180159fe8ada03e03caf5ea1fe68ac2078a25d75b7a9f089ff",
        "domain": "numina",
        "question": "30. [20] Let $A B C$ be a triangle with circumcenter $O$, incenter $I, \\angle B=45^{\\circ}$, and $O I \\| B C$. Find $\\cos \\angle C$.",
        "answer": "1-\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151522,
        "difficulty": null
      },
      {
        "id": "838df4ef1964f20a2768783c3ac519960b0f0b84bfb4065ca75ed785241983f6",
        "domain": "numina",
        "question": "## Problema 4\n\nFie $\\mathrm{A}, \\mathrm{B} \\in M_{2}(\\mathrm{Q})$ astfel încât $\\mathrm{AB}=\\mathrm{BA}$, $\\operatorname{det} \\mathrm{A}=-3$ și $\\operatorname{det}(\\mathrm{A}+\\sqrt{3} \\mathrm{~B})=0$.\n\nSă se calculeze $\\operatorname{det}\\left(A^{2}+B^{2}-A B\\right)$\n\nGMB 12/2011\n\nProbleme selectate de Prof. Ursan Rodica\n\nNotă: a) Timpul efectiv de lucru este de 3 ore.\n\nb) Toate problemele sunt obligatorii.\n\nc) Fiecare problemă se notează de la 0 la 7 .\n\n## OLIMPIADA NAȚIONALĂ DE MATEMATICĂ\n\nEtapa locală - 15.02.2014\n\n## Clasa a XI-a\n\n## Barem de corectare",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 180125,
        "difficulty": null
      },
      {
        "id": "3a25033ad71ef4ae1a89d3c7b7eb04eb1e0f46dbf6c47d6d4d4cfec2497a2959",
        "domain": "numina",
        "question": "9.6. Find the minimum value of the expression $(\\sqrt{2(1+\\cos 2 x)}-\\sqrt{36-4 \\sqrt{5}} \\sin x+2) \\cdot(3+2 \\sqrt{10-\\sqrt{5}} \\cos y-\\cos 2 y) \\cdot$ If the answer is not an integer, round it to the nearest integer.",
        "answer": "-27",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6480,
        "difficulty": null
      },
      {
        "id": "5b176a940e0759be716b4952c9fa2bf8f0874d37e25156510ac42db667ff7ba5",
        "domain": "numina",
        "question": "Task B-2.4. Let $f(x)=x^{2}+b x+c, b, c \\in \\mathbb{R}$. If $f(0)+f(1)=\\frac{1}{2}$, calculate $f\\left(\\frac{1}{2}\\right)$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88886,
        "difficulty": null
      },
      {
        "id": "0a06a17efd01823dc1621fe8d01cb3532ec48e743175970025b6f3359fa39807",
        "domain": "numina",
        "question": "Problem 3. Five different natural numbers are written in a circle on the board. Each of them, Petya divided by the next one in the clockwise direction, and then wrote down the 5 resulting numbers (not necessarily integers) on a piece of paper. Can the sum of the 5 numbers on the paper be an integer?",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4428,
        "difficulty": null
      },
      {
        "id": "7ae58f9a2509045acd5f76d6398b17254672cf03b3a37005387cf808f7a6fc14",
        "domain": "numina",
        "question": "9. Let $p$ and $q$ be integers such that $p+q=2010$. If both roots of the equation $10 x^{2}+p x+q=0$ are positive integers, find the sum of all possible values of $p$.\n(1 mark)\n設 $p 、 q$ 為整數, 且 $p+q=2010$ 。若方程 $10 x^{2}+p x+q=0$ 的兩根均為正整數, 求 $p$ 所有可能值之和。\n（1 分）",
        "answer": "-3100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153619,
        "difficulty": null
      },
      {
        "id": "53f09f11d1e84e58d009296bb0951f13546dba1f181112334c883c78fa46b358",
        "domain": "numina",
        "question": "Example 7. Calculate the area of the figure bounded above by the lines $x-y+2=0, y=2-\\frac{1}{2} x$, and below by the parabola $y=x^{2}-2 x-$ $-8$.",
        "answer": "42",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31269,
        "difficulty": null
      },
      {
        "id": "89484b5c4fc92d7b17ea340559fa3a0528366c6dc434d33f4edc28f3f9a99f65",
        "domain": "numina",
        "question": "5. The zoo keeps some sika deer and ostriches. The number of ostrich feet is 20 more than the number of sika deer feet. When each ostrich lifts one foot, the number of ostrich feet lifted is 10 less than the number of sika deer feet. The zoo has $\\qquad$ sika deer.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109523,
        "difficulty": null
      },
      {
        "id": "e8e4468e37d9008cb47bf3faf704cb44b33bb5a247ee0c320f9c68e927991205",
        "domain": "numina",
        "question": "Problem 11.5. Quadrilateral $ABCD$ is inscribed in a circle. It is known that $BC=CD, \\angle BCA=$ $64^{\\circ}, \\angle ACD=70^{\\circ}$. A point $O$ is marked on segment $AC$ such that $\\angle ADO=32^{\\circ}$. How many degrees does the angle $BOC$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-43.jpg?height=448&width=519&top_left_y=234&top_left_x=465)",
        "answer": "58",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13453,
        "difficulty": null
      },
      {
        "id": "c84c40a05a0f3d953e67e099f72d19c52da6f27993b00c4ff482c7cd9f30c603",
        "domain": "numina",
        "question": "4. In the group, there are 6 Varaždinians, 6 Zadrians, and 6 Osijekians. In how many ways can they be divided into:\na) 6 groups of 3 people so that each group contains one Varaždinian, one Zadrian, and one Osijekian?\nb) 3 groups of 6 people so that each group contains two Varaždinians, two Zadrians, and two Osijekians?",
        "answer": "518400",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 86597,
        "difficulty": null
      },
      {
        "id": "c5a9dd919aa25b925577e8fee1104d88f03b00de60b3831d91427724b75e17d4",
        "domain": "numina",
        "question": "135. A troop of monkeys was having fun; the square of one eighth of them was frolicking in the forest, the remaining twelve were shouting at the top of a hillock. Tell me: how many monkeys were there in total?",
        "answer": "4816",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 37777,
        "difficulty": null
      },
      {
        "id": "4bb877f12dc45551aa6366b455b61220884593ad003fe3e4e53536fd3be8b46a",
        "domain": "numina",
        "question": "1. Find the number of triples $(a, b, c)$ of positive integers such that $a+a b+a b c=11$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151580,
        "difficulty": null
      },
      {
        "id": "ae9c678b793ca0c5ea13e4eb0cf970367926d723e44e862fd9d2f3a5166f30cb",
        "domain": "numina",
        "question": "$:$ Folklore\n\nLёsha thought of a two-digit number (from 10 to 99). Grisha tries to guess it by naming two-digit numbers. If Grisha correctly names the number, or one digit is correct and the other is off by no more than one, Lёsha responds with \"warm\"; in all other cases, Lёsha responds with \"cold\". (For example, if the number 65 is thought of, then by naming 65, 64, 66, 55, or 75, Grisha will hear \"warm,\" and in all other cases, \"cold.\")\n\na) Show that there is no way for Grisha to guarantee finding the number using 18 attempts.\n\nb) Come up with a way for Grisha to guarantee finding the number using 24 attempts (regardless of which number Lёsha thinks of).\n\nc) Can it be done in 22 attempts?",
        "answer": "24",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 65983,
        "difficulty": null
      },
      {
        "id": "c331b5ae5ba6010b6df4384985afd33f80b9fcfd726a9a8c8c577ba7761f58a5",
        "domain": "numina",
        "question": "9. Circles $\\omega_{a}, \\omega_{b}, \\omega_{c}$ have centers $A, B, C$, respectively and are pairwise externally tangent at points $D, E, F$ (with $D \\in B C, E \\in C A, F \\in A B$ ). Lines $B E$ and $C F$ meet at $T$. Given that $\\omega_{a}$ has radius 341 , there exists a line $\\ell$ tangent to all three circles, and there exists a circle of radius 49 tangent to all three circles, compute the distance from $T$ to $\\ell$.",
        "answer": "294",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151836,
        "difficulty": null
      },
      {
        "id": "591b983a3d589d38b4f86986386084a43b9d6b3385e506ce8762d159a1353a2b",
        "domain": "numina",
        "question": "5. In the Cartesian coordinate system $x O y$, there are 50 different parabolas $y=a x^{2}+b x+c$ and another 50 different parabolas $x=a_{1} y^{2}+b_{1} y+c_{1}$. These 100 parabolas divide the coordinate plane into a maximum of $\\qquad$ parts.",
        "answer": "15052",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 107302,
        "difficulty": null
      },
      {
        "id": "e3d8a962e0dcb5c422c62d27b6f631e6d41e799038a0cd1ba90e25ae8bde3d4a",
        "domain": "numina",
        "question": "## Task 4 - 271214\n\nDetermine the remainder when the sum $s=1^{5}+2^{5}+3^{5}+\\ldots+1987^{5}$ is divided by 25!",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 186134,
        "difficulty": null
      },
      {
        "id": "bcf2d1af802bc19837db387ad51ab57a1353c67fe5a222782c81359ee4e95001",
        "domain": "numina",
        "question": "Example 12 The sports meet lasted for $n$ days $(n>1)$, and a total of $m$ medals were awarded. On the first day, 1 medal was awarded, and then $\\frac{1}{7}$ of the remaining $m-1$ medals. On the second day, 2 medals were awarded, and then $\\frac{1}{7}$ of the remaining medals, and so on, until on the $n$-th day exactly $n$ medals were awarded, and all were distributed. How many days did the sports meet last? How many medals were awarded in total?\n(IMO - 9 problem)",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144919,
        "difficulty": null
      },
      {
        "id": "d3d20e669bd778cca110df242053877c4d9390b21440e3ba89b8709460558b07",
        "domain": "numina",
        "question": "5. AFS International Cultural Exchange Organization plans to allocate 18 slots for a high school exchange program to 4 schools, with the requirement that each school gets at least one slot and the number of slots allocated to each school must be different. The number of different allocation methods is $\\qquad$ .",
        "answer": "360",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 93321,
        "difficulty": null
      },
      {
        "id": "1f9257d95bc313de321e51927a53b688a28c7ae51739d13ad3d2831318b54f26",
        "domain": "numina",
        "question": "Example 11 (23rd All-Union Mathematical Olympiad, 1989) A natural number $N$ has exactly 12 positive divisors (including 1 and $N$), which are numbered in increasing order: $d_{1}<d_{2}<\\cdots<d_{12}$. It is known that the positive divisor with index $d_{4}-1$ is equal to $\\left(d_{1}+d_{2}+d_{4}\\right) \\cdot d_{8}$. Find the natural number $N$.",
        "answer": "1989",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105934,
        "difficulty": null
      },
      {
        "id": "2480a483e23bc0033385f76a72176377b04e39ab665018e89046d225a7f6d6a5",
        "domain": "numina",
        "question": "146. In a regular quadrilateral pyramid, the center of the circumscribed sphere lies on the surface of the inscribed sphere. Find the measure of the plane angle at the vertex of the pyramid.",
        "answer": "\\frac{\\pi}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29852,
        "difficulty": null
      },
      {
        "id": "4d1c8a5cc94bee2549c2b3ccbcae2731ee1f88f64961f9a67377810b3ba5d138",
        "domain": "numina",
        "question": "5. Each football is sewn from pieces of leather in the shape of regular pentagons and regular hexagons. The ball has a total of 32 leather pieces. Each piece of leather in the shape of a pentagon is connected along its sides only to pieces of leather in the shape of hexagons. Each piece of leather in the shape of a hexagon is connected along its sides to 3 pentagons and 3 hexagons. How many pieces of leather are in the shape of a pentagon, and how many in the shape of a hexagon?",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189232,
        "difficulty": null
      },
      {
        "id": "5ede8c550947b7e663fca19ade9729c99c04ed58a73c875d1458de69d8b77347",
        "domain": "numina",
        "question": "9-1. Segment $P Q$ is divided into several smaller segments. On each of them, a square is constructed (see figure).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7c4dd4d2d6040292f233g-20.jpg?height=619&width=1194&top_left_y=593&top_left_x=431)\n\nWhat is the length of the path along the arrows if the length of segment $P Q$ is 73? If necessary, round the answer to 0.01 or write the answer as a common fraction.",
        "answer": "219",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13664,
        "difficulty": null
      },
      {
        "id": "65867267e5b1cf4f428a91a61d3a253bb860959338b13ef023f880c1deaed3e5",
        "domain": "numina",
        "question": "Given a regular tetrahedron $S-ABC$ with height $SO=3$, and the side length of the base is 6. A perpendicular line is drawn from point $A$ to the opposite face $SBC$, with the foot of the perpendicular being $O'$. On $AO'$, take a point $P$ such that $\\frac{AP}{PO'}=8$. Find the area of the section passing through point $P$ and parallel to the base.",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 135254,
        "difficulty": null
      },
      {
        "id": "40252f26da1a34aecb680824d1ff4a17498c620a70239b5f3474c48a21e80ba3",
        "domain": "numina",
        "question": "$7 \\cdot 115$ Let $S=\\{1,2,3,4\\}, a_{1}, a_{2}, \\cdots, a_{k}$ be a sequence composed of numbers from $S$, and it includes all permutations of $S$ that do not end with 1, i.e., for any permutation $\\left(b_{1}, b_{2}, b_{3}, b_{4}\\right)$ of the 4 numbers in $S$, where $b_{4} \\neq 1$, there exist $i_{1}, i_{2}, i_{3}, i_{4}$, such that\n$$\na_{i_{j}}=b_{j}, j=1,2,3,4 \\text { and } 1 \\leqslant i_{1}<i_{2}<i_{3}<i_{4} \\leqslant k .\n$$\n\nFind the minimum value of the number of terms $k$ in the sequence.",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96726,
        "difficulty": null
      },
      {
        "id": "f4712b44027a4e8a5c8bb9be1ec50a2b19e22852da4da8aa06b5f0c29a0c2522",
        "domain": "numina",
        "question": "11.1. In triangle $\\mathrm{ABC}$, the bisectors of angles $\\mathrm{BAC}$ and $\\mathrm{BCA}$ intersect sides ВС and АВ at points К and Р, respectively. It is known that the length of side АС is equal to the sum of the lengths of segments АР and СК. Find the measure of angle $\\mathrm{ABC}$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 9976,
        "difficulty": null
      },
      {
        "id": "507e8de96c411bae45c09d7bab91c0e1b2e7430f6f194dc7c3e2f5b61f005d52",
        "domain": "numina",
        "question": "9.2. In triangle $ABC$, the median $BM$ was drawn. It turned out that $\\angle ABM=40^{\\circ}$, $\\angle MBC=70^{\\circ}$. Find the ratio $AB: BM$. Justify your answer.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_94f1ab38259b8995cf4bg-4.jpg?height=494&width=708&top_left_y=1763&top_left_x=137)\n\nTo solve problem 9.2 Solution: Extend the median $BM$ beyond point $M$ and lay off a segment $MD = BM$ (see the figure). In quadrilateral $ABCD$, the diagonals bisect each other, so this quadrilateral is a parallelogram. Then $\\angle ABC + \\angle BCD = 180^{\\circ}$, from which $\\angle BCD = 70^{\\circ} = \\angle MBC = \\angle DBC$. This means that triangle $BCD$ is isosceles, as is the congruent triangle $BAD$. Therefore,\n\n$$\nAB: BM = AB: \\frac{BD}{2} = AB: \\frac{AB}{2} = 2\n$$",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 16642,
        "difficulty": null
      },
      {
        "id": "e2b0a64cd806f1b7cc78f9803cff8365d124bf5c14697b7d2fabe848a9a7e61a",
        "domain": "numina",
        "question": "7. Given in $\\triangle A B C$, $a=2 b, \\cos B=\\frac{2 \\sqrt{2}}{3}$, then $\\sin \\frac{A-B}{2}+\\sin \\frac{C}{2}=$ $\\qquad$",
        "answer": "\\frac{\\sqrt{10}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119324,
        "difficulty": null
      },
      {
        "id": "bb14d5aea08e8ffb8e177001c77a58b084b3dd5651067433fbe5c87b1ac81b7c",
        "domain": "numina",
        "question": "[ Trigonometric ratios in a right triangle [ Theorem of the length of a tangent and a secant; the product of the entire secant and its external part [ Two tangents drawn from the same point\n\nIn parallelogram $K L M N$, side $K L$ is equal to 8. A circle, tangent to sides $N K$ and $N M$, passes through point $L$ and intersects sides $K L$ and $M L$ at points $C$ and $D$ respectively. It is known that $K C: L C=4: 5$ and $L D: M D=8: 1$. Find side $K N$.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61421,
        "difficulty": null
      },
      {
        "id": "d665142dab10ba571270525b16c9a2672a4a70189e129903bb1b602173bfa8ac",
        "domain": "numina",
        "question": "10.1. Different positive numbers $x, y, z$ satisfy the equations\n\n$$\nx y z=1 \\quad \\text{and} \\quad x+y+z=\\frac{1}{x}+\\frac{1}{y}+\\frac{1}{z}\n$$\n\nFind the median of them. Justify your answer.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13295,
        "difficulty": null
      },
      {
        "id": "3f9935a326309c7d66b1a2751769028090e2aa2e4523ef611657674c69641040",
        "domain": "numina",
        "question": "$5 \\cdot 71$ A city has $n$ high schools, the $i$-th high school sends $C_{i}$ students to the stadium to watch a ball game. It is known that $0 \\leqslant C_{i} \\leqslant 39, i=1,2, \\cdots, n, C_{1}+C_{2}+\\cdots+C_{n}=1990$, each row of seats in the stand has 199 seats. It is required that students from the same school must sit in the same row. How many rows at least should the stadium arrange to ensure that all students can be seated according to the requirements?",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 95543,
        "difficulty": null
      },
      {
        "id": "9b5ce99e9c2e561e98f7c2c02b6dddadb52990e23eb6d9f90a4ca536623e7333",
        "domain": "numina",
        "question": "10. If $x=\\frac{1}{2}\\left(\\sqrt[3]{2009}-\\frac{1}{\\sqrt[3]{2009}}\\right)$, find the value of $\\left(x+\\sqrt{1+x^{2}}\\right)^{3}$.",
        "answer": "2009",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160526,
        "difficulty": null
      },
      {
        "id": "edfc8f229a30d7ed561519fadbc15e084cd09e364560e77d459accbd3fa0208a",
        "domain": "numina",
        "question": "2. Tine was at a competition where he was supposed to solve 20 problems. For each correctly solved problem, he received 8 points, and for each incorrect solution, 5 points were deducted. For a problem he did not attempt, he received 0 points. He found out that he had scored 13 points. How many problems did he attempt?",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162567,
        "difficulty": null
      },
      {
        "id": "b1be5ea5ae820b2e9e2d065ced80caf404642d0f46f3b4ea5e584b86e3c21f8c",
        "domain": "numina",
        "question": "16. A polygon is said to be friendly if it is regular and it also has angles that when measured in degrees are either integers or half-integers (i.e. have a decimal part of exactly 0.5 ). How many different friendly polygons are there?",
        "answer": "28",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156326,
        "difficulty": null
      },
      {
        "id": "27998c5d8d0f132fe52b3449a190d9e317c4fe7e7a57137a16ac4da03de30665",
        "domain": "numina",
        "question": "7. (10 points) Uncle Cook's hat fell in front of the gate, still smoking. It turned out that someone had thrown a firecracker from the window, and the falling firecracker had set the hat on fire. When the accident happened, 5 boys all poked their heads out, and of course, none of them were willing to admit it. Now, 4 of the boys are telling the truth, and one person is lying, and the liar is the one who threw the firecracker. So, the liar's room number is $\\qquad$Buster: “It wasn't me, I only found out what happened when Uncle Cook shouted.” Oak: “It wasn't me, Marco can vouch for me, I didn't throw anything.” Marco: “It wasn't Oak, it didn't come from above, I didn't see anything, and I didn't throw anything.” Corobi: “But I saw it, someone threw something from above.”\n\nMarv: “Yes, someone threw something from above, it flew over my head, just grazing my hair.”",
        "answer": "302",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 109124,
        "difficulty": null
      },
      {
        "id": "30f80ce1aa3dffd50732547332d04c9c4f1dfd9d79267760375acfbce8567483",
        "domain": "numina",
        "question": "3. For a given positive constant $k$, define $f_{1}(k)$ as the square of the sum of the digits of $k$, and let $f_{n+1}(k)=f_{1}\\left(f_{n}(k)\\right)$, find the value of $f_{1991}\\left(2^{1990}\\right)$.\n(IMO - 31 Shortlist)",
        "answer": "169",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 91905,
        "difficulty": null
      },
      {
        "id": "aff39edf90a22e4204a90ab80f8cba06c227a41077ffbc0c6a25f091411b9c54",
        "domain": "numina",
        "question": "4.1. (14 points) How many six-digit numbers exist whose sum of digits equals 51?",
        "answer": "56",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 6181,
        "difficulty": null
      },
      {
        "id": "b327d15b68b7c683ea389ff683677812b410f968fa2c8e59f975f12ba4ae3800",
        "domain": "numina",
        "question": "On the number line, a flea is jumping. It starts from 0, the length of each jump is 1, and the next jump is in the same direction as the previous one with probability $p$, and in the opposite direction with probability $1-p$. What is the probability that it will return to 0?",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 170015,
        "difficulty": null
      },
      {
        "id": "74cee1a0e6c3694e1fea41ef0bfff80b78e59166ef891af0981a1349aecefa60",
        "domain": "numina",
        "question": "1. In a $4 \\times 4$ table, positive real numbers are written such that the product in each pentad of the shape 母 is equal to 1. Determine the maximum number of different numbers written in the table.\n\nNote: The shape 母 refers to a specific arrangement of five cells in the table, resembling the Chinese character 母. For clarity, it typically involves a central cell surrounded by four adjacent cells.",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 75913,
        "difficulty": null
      },
      {
        "id": "cccdfddff1babedee293de91931483d8ed5aad6f0a1eb15754579d2f5dd88bb3",
        "domain": "numina",
        "question": "8. Let $M=\\{1,2, \\cdots, 2017\\}$ be the set of the first 2017 positive integers. If one element is removed from the set $M$, and the sum of the remaining elements is exactly a perfect square, then the removed element is",
        "answer": "1677",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122575,
        "difficulty": null
      },
      {
        "id": "376f46b0c7d6d39f9736b6098a0a6d885d12f0fe922f277523e72f322baf72e6",
        "domain": "numina",
        "question": "18 Given a regular triangular prism $A B C-A_{1} B_{1} C_{1}$, where $A A_{1}=2 A C=4$, extend $C B$ to $D$ such that $C B=B D$.\n(1) Prove that line $C_{1} B$ // plane $A B_{1} D$;\n(2) Find the sine value of the angle formed by plane $A B_{1} D$ and plane $A C B$.",
        "answer": "\\frac{4\\sqrt{17}}{17}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 132637,
        "difficulty": null
      },
      {
        "id": "6a1d1e1a9471aa18cfb2c8b873b7f43ec9cd17f42cc3177fc202c5987e83107b",
        "domain": "numina",
        "question": "[ The angle between two chords and two secants ] [ The inscribed angle is half of the central angle ]\n\nChords $A C$ and $B D$ are drawn from the ends of arc $A B$, which contains $m^{\\circ}$, such that the angle $D M C$, formed by their intersection, is equal to the inscribed angle $D N C$ in arc $C D$. Find the degree measure of this arc.",
        "answer": "180-\\frac{}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59597,
        "difficulty": null
      },
      {
        "id": "e120a468d2d00d2953ee2fbd6e1565bf0d7482d90c83f560b8baac9d74f6bd61",
        "domain": "numina",
        "question": "3. Given a convex $n$-sided polygon, each of its diagonals is the perpendicular bisector of at least one other diagonal. Find all possible positive integers $n(n>3)$.\n\n untranslated text remains in its original format and line breaks are preserved.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 99104,
        "difficulty": null
      },
      {
        "id": "fa961049cd05af32b1c878b6406acc3c9b3e000bb7e79b61c72e7c0c7b432752",
        "domain": "numina",
        "question": "44th Putnam 1983 Problem B1 Let C be a cube side 4, center O. Let S be the sphere center O radius 2. Let A be one of the vertices of the cube. Let R be the set of points in C but not S, which are closer to A than to any other vertex of C. Find the volume of R. Solution",
        "answer": "8-\\frac{4\\pi}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 148583,
        "difficulty": null
      },
      {
        "id": "05c993d4e13464c01e85458fa891d0c0701e8dcbf763a72bfb28c50c77813b19",
        "domain": "numina",
        "question": "12. The mid-term exam score statistics for Chinese, Mathematics, and English in a class are as follows: 18 students have at least one subject with an excellent grade; 9 students have an excellent grade in Chinese, 11 students have an excellent grade in Mathematics, and 8 students have an excellent grade in English; 5 students have an excellent grade in both Chinese and Mathematics, 3 students have an excellent grade in both Mathematics and English, and 4 students have an excellent grade in both Chinese and English. Ask:\n(1) How many students have at least one excellent grade in Chinese or Mathematics?\n(2) How many students have an excellent grade in all three subjects: Chinese, Mathematics, and English?",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 103918,
        "difficulty": null
      },
      {
        "id": "d8b32864cd5f3e193b00a112660f1744f8b800edaf566dabce04f04b5c069f9e",
        "domain": "numina",
        "question": "2. There is a sequence of numbers, starting from the 3rd number, each number is the sum of all the numbers before it. If the 1st number of this sequence is 1, and the 2nd number is 2, then the remainder when the 2022nd number is divided by 7 is $\\qquad$ -",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 117911,
        "difficulty": null
      },
      {
        "id": "bdf6a63f1e7205e85708bbcc8b95d0860195bac708c5ba34a499ce81542a970b",
        "domain": "numina",
        "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$$\n\\begin{aligned}\n& M_{1}(14 ; 4 ; 5) \\\\\n& M_{2}(-5 ;-3 ; 2) \\\\\n& M_{3}(-2 ;-6 ;-3) \\\\\n& M_{0}(-1 ;-8 ; 7)\n\\end{aligned}\n$$",
        "answer": "3\\sqrt{\\frac{13}{2}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 46265,
        "difficulty": null
      },
      {
        "id": "3e599be98013f15522117db7ca1e84942306db6d1923dc6c7ee52d5d43b279fa",
        "domain": "numina",
        "question": "## Task 1\n\nThe fifteenfold of a number is greater than 299 and less than 301.\n\na) Write the information as an inequality.\n\nb) What is the number?",
        "answer": "20",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 186763,
        "difficulty": null
      },
      {
        "id": "3efeb1591ee47d69302208eb1e724a2756902c32444e772b3977a236cbab71cb",
        "domain": "numina",
        "question": "14. $A B C D$ is a square with side length 9 . Let $P$ be a point on $A B$ such that $A P: P B=7: 2$. Using $C$ as centre and $C B$ as radius, a quarter circle is drawn inside the square. The tangent from $P$ meets the circle at $E$ and $A D$ at $Q$. The segments $C E$ and $D B$ meet at $K$, while $A K$ and $P Q$ meet at $M$. Find the length of $A M$.\n$A B C D$ 是一個邊長爲 9 的正方形。設 $P$ 爲 $A B$ 上的一點, 使得 $A P: P B=7: 2 \\circ$ 以 $C$ 緺圓心 、 $C B$ 爲牛徑在正方形內作一個四分一圓, 從 $P$ 點到圓的切線與圓相交於 $E$, 與 $A D$ 相交於 $Q$ 。 $C E$ 和 $D B$ 交於 $K$, 而 $A K$ 和 $P Q$ 則交於 $M$ 。求 $A M$ 的長度。",
        "answer": "\\frac{85}{22}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153541,
        "difficulty": null
      },
      {
        "id": "60b5a941a9f4c141c32b82dfd47e40149d88090bd7eb5264bdd0c82a78b51dfb",
        "domain": "numina",
        "question": "2. Determine all positive integers that are equal to 300 times the sum of their digits.",
        "answer": "2700",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 181678,
        "difficulty": null
      },
      {
        "id": "b73f10b230dd6b1261c17e993eab0ca162e586865c475b46a41be9b7a6773c2b",
        "domain": "numina",
        "question": "3. The area of the closed figure enclosed by the curve $x^{2}+2 y^{2} \\leqslant 4|y|$ is . $\\qquad$",
        "answer": "2\\sqrt{2}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105075,
        "difficulty": null
      },
      {
        "id": "6b59ca91db31f84f62874a4cdbaed93535b32a59fef3a48767061b4659e0ae83",
        "domain": "numina",
        "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4984b677816e04f7f1c9g-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
        "answer": "43",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14837,
        "difficulty": null
      },
      {
        "id": "9e57e5db96ca9414ec1f3e0104287fe40bda5bdc467af9838d57a757168af858",
        "domain": "numina",
        "question": "B4. Calculate exactly, without using a pocket calculator: $\\frac{(-2)^{-3}}{(-0.2)^{3}}-\\left(\\frac{2}{5}\\right)^{-3} \\cdot(-3)^{-2} \\cdot 0.1^{-1}$.\n\n## PROBLEMS FOR THE THIRD YEAR\n\nBefore you are two sets of problems. Problems 1 to 6 of the first set are solved by selecting the correct answer from the five proposed answers and writing it in the table below the corresponding number. Only one answer is correct. A correct answer will be scored with two points, while a wrong answer will result in the deduction of one point.\n\nProblems 1 to 4 of the second set are solved on the attached paper. The solution to each of these problems will be evaluated on a scale from 0 to 6 points. Write only your code on the sheets where you will solve the problems. Write your solution clearly and legibly with a pen.\n\nThe time allowed for solving is 90 minutes.\n\nTHE NATIONAL COMPETITION COMMITTEE WISHES YOU GREAT SUCCESS.\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_ce887fced8b2a2bf0894g-06.jpg?height=168&width=485&top_left_y=904&top_left_x=800)\n\n## PART I",
        "answer": "-\\frac{125}{72}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163906,
        "difficulty": null
      },
      {
        "id": "9212c93397fe19849036193aa6a19dd58f4ecd6304ec066282187e64ab20e26f",
        "domain": "numina",
        "question": "B2 The point $S$ lies on the chord $A B$ of a circle such that $S A=3$ and $S B=5$. The radius of the circle from the center $M$ through $S$ intersects the circle at $C$. Given $C S=1$.\n\nCalculate the length of the radius of the circle.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_3de5ef26943dd9bef1e9g-2.jpg?height=411&width=640&top_left_y=1279&top_left_x=1116)",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 85792,
        "difficulty": null
      },
      {
        "id": "643fc02d8cf983e2a6405b78aaf916cbdd6855718040e637ccb69df0b4b306e9",
        "domain": "numina",
        "question": "[ equations in integers ]\n\nThe number 1047, when divided by $A$, gives a remainder of 23, and when divided by $A+1$, it gives a remainder of 7. Find $A$.\n\n#",
        "answer": "64",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68512,
        "difficulty": null
      },
      {
        "id": "9a185658e51ee211b724c1d1a2334d7c2b2567f87838efd41836174150030582",
        "domain": "numina",
        "question": "9.4 In the thirtieth kingdom, there are three types of coins in circulation: bronze rubles, silver coins worth 9 rubles, and gold coins worth 81 rubles. From the treasury, which contains an unlimited supply of each type of coin, a certain amount was issued with 23 coins, which is less than 700 rubles. Find this amount, given that it is impossible to issue it with a smaller number of coins.",
        "answer": "647",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12935,
        "difficulty": null
      },
      {
        "id": "7aad72226a780bf82da9d1a3ed4dce0b1df81bf0eb47758b8ac74fc60932a3b3",
        "domain": "numina",
        "question": "9.6. There are 2013 cards with the digit 1 written on them, and 2013 cards with the digit 2 written on them. Vasya forms a 4026-digit number from these cards. In one move, Petya can swap two cards and pay Vasya 1 ruble. The process ends when Petya gets a number divisible by 11. What is the maximum amount Vasya can earn if Petya aims to pay as little as possible?\n\n(P. Kozhevnikov)",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 14159,
        "difficulty": null
      },
      {
        "id": "a168c08a49455a9dddb74e76db8807f4770540354b05e86a1383c3bd6a6018f9",
        "domain": "numina",
        "question": "The sequence of numbers $t_{1}, t_{2}, t_{3}, \\ldots$ is defined by\n\n$$\n\\left\\{\\begin{array}{l}\nt_{1}=2 \\\\\nt_{n+1}=\\frac{t_{n}-1}{t_{n}+1}\n\\end{array}\\right.\n$$\n\nSuggestion: Calculate the first five terms of the sequence.\n\nfor each positive integer $n$. Find $\\mathrm{t}_{2011}$.",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74514,
        "difficulty": null
      },
      {
        "id": "4f9a103afb1e71d7fa26803aaaad034247d9a96302e1a6d287a74a592a01df34",
        "domain": "numina",
        "question": "6.68. Using Taylor and Maclaurin formulas, compute the limits\n\na) $\\lim _{x \\rightarrow 0} \\frac{\\sin x-x}{x^{3}}$\n\nb) $\\lim _{x \\rightarrow 0} \\frac{\\ln (1+x)-x+x^{2} / 2}{x^{3}}$.",
        "answer": "\\frac{1}{6}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33419,
        "difficulty": null
      },
      {
        "id": "f0d9ffc386d0c39573b8bc3977eed88ccf9658aebab0b570d54b37113c485f84",
        "domain": "numina",
        "question": "(3) Let $M$ be a set of $n$ points in the plane, satisfying:\n(1) There exist 7 points in $M$ that are the 7 vertices of a convex heptagon;\n(2) For any 5 points in $M$, if these 5 points are the 5 vertices of a convex pentagon, then this convex pentagon contains at least one point from $M$ inside it. Find the minimum value of $n$.",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 105042,
        "difficulty": null
      },
      {
        "id": "f8abff4489dd4cfa112eec6db2182396e7f96494bdecf66e2a29a541e45cd990",
        "domain": "numina",
        "question": "The angles of a triangle are $\\alpha, \\beta, \\gamma$. What is the maximum possible value of\n\n$$\n\\sin \\alpha \\sin \\beta \\cos \\gamma+\\sin ^{2} \\gamma\n$$",
        "answer": "\\frac{9}{8}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 169957,
        "difficulty": null
      },
      {
        "id": "837f761d020e3aadcfd58aa541f31a0f3a78401abc5614f01f9588f370417069",
        "domain": "numina",
        "question": "47. Find the limit of the function:\n1) $f(x)=x^{3}-5 x^{2}+2 x+4$ as $x \\rightarrow-3$;\n2) $\\varphi(t)=t \\sqrt{t^{2}-20}-\\lg \\left(t+\\sqrt{t^{2}-20}\\right)$ as $t \\rightarrow 6$.",
        "answer": "23",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 34189,
        "difficulty": null
      },
      {
        "id": "1425023d5cb88ce2da05dd154331326c6008bbf2748d951a49c5a2aa8bb636bb",
        "domain": "numina",
        "question": "8. (10 points) Three households, A, B, and C, plan to subscribe to newspapers. There are 7 different newspapers available. It is known that each household subscribes to 3 different newspapers, and any two households have exactly one newspaper in common. Therefore, the three households have $\\qquad$ different subscription methods.",
        "answer": "5670",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 107393,
        "difficulty": null
      },
      {
        "id": "23dc6a0bdb6d7db21338d71a67e4fb8e70aa636d2d0d461355d30f289526c141",
        "domain": "numina",
        "question": "Task B-1.7. Ivan and Nenad collect 2-kuna and 5-kuna coins in their respective piggy banks. After a year, they opened their piggy banks and counted the coins. Together, they collected 280 2-kuna coins. In Ivan's piggy bank, $60\\%$ of all the coins are 2-kuna coins, and in Nenad's piggy bank, there are twice as many 2-kuna coins as 5-kuna coins. The total value of the money from both piggy banks together is 1360 kuna. What is the value of the money from Ivan's piggy bank, and what is the value from Nenad's?",
        "answer": "640",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89305,
        "difficulty": null
      },
      {
        "id": "ee96106c3fd042780bdbfed3a074728b5c7eb0edac3ff49cb97387b609d49c56",
        "domain": "numina",
        "question": "4. (8 points) In the following horizontal equation, the same Chinese characters represent the same digits, different Chinese characters represent different digits, and no Chinese character represents 7. \"迎\", \"春\", and \"杯\" are not equal to 1. Therefore, the sum of the three digits represented by \"迎\", \"春\", and \"杯\" is $\\qquad$\nLearn math $\\times 7 \\times$ Welcome $\\times$ Spring $\\times$ Cup $=$ Cheer up cheer up $\\times$ Bar.",
        "answer": "15",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 100281,
        "difficulty": null
      },
      {
        "id": "d2db98215fe99428953dcf485088c813523c9c5f107a1ff51421d79218bc6fa1",
        "domain": "numina",
        "question": "5. By definition, $n!=1 \\cdot 2 \\cdot 3 \\cdot \\ldots \\cdot n$. Which factor should be removed from the product $1! \\cdot 2! \\cdot 3! \\cdot \\ldots \\cdot 20!$ so that the remaining product becomes a square of some natural number?",
        "answer": "10!",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17907,
        "difficulty": null
      },
      {
        "id": "16e68281d829659ac240f439d880dd17ccdfd6cd9181994573b3a3b86d40606b",
        "domain": "numina",
        "question": "8-1. Kolya made a figure from four identical blocks as shown in the picture. What is the surface area of this figure? Express your answer in square centimeters.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1421239a8d2e12a5ec5fg-01.jpg?height=450&width=1295&top_left_y=786&top_left_x=355)",
        "answer": "64",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15721,
        "difficulty": null
      },
      {
        "id": "e66134b91dda84062ad900dc536f3d04ca5ae2ca259804666a3ad3d86fae9a86",
        "domain": "numina",
        "question": "Problem 2. In a square of grid paper containing an integer number of cells, a hole in the shape of a square, also consisting of an integer number of cells, was cut out. How many cells did the large square contain if 209 cells remained after the cutout?",
        "answer": "225",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 2802,
        "difficulty": null
      },
      {
        "id": "065316757752f253ab74a21919f2f774ca701f89e5ead0644d44c23e8eaeeff9",
        "domain": "numina",
        "question": "4. Given a tetrahedron $P-ABC$ with three lateral edges $PA, PB, PC$ that are pairwise perpendicular, let the dihedral angles $P-AB-C, P-BC-A$, $P-CA-B$ be $\\alpha, \\beta, \\gamma$ respectively,\nthen $\\frac{\\sin ^{2} \\alpha+\\sin ^{2} \\beta+\\sin ^{2} \\gamma}{\\cos ^{2} \\alpha+\\cos ^{2} \\beta+\\cos ^{2} \\gamma}=$ $\\qquad$ .",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 121108,
        "difficulty": null
      },
      {
        "id": "7eee5325139efba8326a98dd193a36c9de418aedc9b71ee8c5d890002a40dac3",
        "domain": "numina",
        "question": "6. (10 points) $n$ pieces of paper, each written with 3 different positive integers not greater than $n$, any 2 pieces of paper have exactly one number in common, find the sum of all numbers written on the pieces of paper.",
        "answer": "84",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 100112,
        "difficulty": null
      },
      {
        "id": "d12c8a9f22bb0557d771657125d0fb34e688502691b7fe5316ffa3491432385f",
        "domain": "numina",
        "question": "João has, in his garden, a cistern where he stores rainwater and takes water to water his flowers. At midnight on December 31, 2005, the cistern contained 156 liters of water. João has the habit of writing on a board, every day, the number of liters of water he spends to water the flowers and the amount of rainwater collected.\n\nBelow we see part of the board referring to the first 8 days of January 2006. How many liters of water were in João's cistern at midnight on January 8, 2006?\n\n| Jan | flowers(l) | rain(l) |\n| :---: | :---: | :---: |\n| 1 | 6 | 2.5 |\n| 2 | 9 | 0 |\n| 3 | 0 | 5 |\n| 4 | 4 | 0 |\n| 5 | 9 | 3 |\n| 6 | 0 | 0 |\n| 7 | 11 | 4.5 |\n| 8 | 0 | 0 |",
        "answer": "132",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74846,
        "difficulty": null
      },
      {
        "id": "9fee198e32bafb3ebc58b2b237602fbeb262237c154d8812634f3db40deeaa63",
        "domain": "numina",
        "question": "Variant 9.2.4. On the sides $AB$ and $AD$ of rectangle $ABCD$, points $M$ and $N$ are marked, respectively. It is known that $AN=9$, $NC=39$, $AM=10$, $MB=5$.\n\n(a) (1 point) Find the area of rectangle $ABCD$.\n\n(b) (3 points) Find the area of triangle $MNC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f4981519479532effc1g-10.jpg?height=428&width=603&top_left_y=1134&top_left_x=419)",
        "answer": "675",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17258,
        "difficulty": null
      },
      {
        "id": "1a99599bbd82d90665043225f3c045daf99a40694622f5f562fe2c7bb4d0b73a",
        "domain": "numina",
        "question": "Problem 5. A group of tourists is dividing cookies. If they evenly distribute two identical packs, one extra cookie will remain. But if they evenly distribute three such packs, 13 extra cookies will remain. How many tourists are in the group? [7 points] (I.V. Raskina)",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 22852,
        "difficulty": null
      },
      {
        "id": "a92118d93fd7fa8a38f3a5f48010d9e993942c344fb305849173d007809e667a",
        "domain": "numina",
        "question": "5-0. A certain quadratic trinomial $x^{2}-p x+q$ has integer roots $x_{1}$ and $x_{2}$. It turns out that the numbers $x_{1}$, $x_{2}$, and $q$ form a decreasing arithmetic progression. Find the sum of all possible values of $x_{2}$.",
        "answer": "-5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15112,
        "difficulty": null
      },
      {
        "id": "ecd32982c05e635ca31349ac785f455779476d61b3611900727090dd00cec88f",
        "domain": "numina",
        "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$M_{1}(1 ; 3 ; 6)$\n\n$M_{2}(2 ; 2 ; 1)$\n\n$M_{3}(-1 ; 0 ; 1)$\n\n$M_{0}(5 ;-4 ; 5)$",
        "answer": "2\\sqrt{14}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 46530,
        "difficulty": null
      },
      {
        "id": "52eef7106855a66c693c3a5523ce3b6d203add4ca135e98f190c1e0717f4d4d0",
        "domain": "numina",
        "question": "Example 17 (1998 \"Hope Cup\" National Invitational Competition for Senior High School Training Question) Given $x, y, z \\in \\mathbf{R}^{+}$, find the maximum value of $\\frac{x y + y z}{x^{2} + y^{2} + z^{2}}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117692,
        "difficulty": null
      },
      {
        "id": "d52682df4b91aa7bb0ccf7daa667264f670578c6cf5cc6980722bba6893a270f",
        "domain": "numina",
        "question": "(8) Let $p$ and $q$ be two different prime numbers, then the remainder when $p^{q-1}+q^{p-1}$ is divided by $p \\cdot q$ is $\\qquad$",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122122,
        "difficulty": null
      },
      {
        "id": "d33b6b9f61063d0292361b998cb5b20eda5e2f9dd9e66503b15501bd2013e450",
        "domain": "numina",
        "question": "31st IMO 1990 shortlist Problem 9 ABC is a triangle with incenter I. M is the midpoint of AC and N is the midpoint of AB. The lines NI and AC meet at X, and the lines MI and AB meet at Y. The triangles AXY and ABC have the same area. Find angle A. Solution Solution by Vivek Kumar Mehra",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150822,
        "difficulty": null
      },
      {
        "id": "01d76b41ed909cd0677d48d0a5da3b536a66443a6658d68697ddb34ce2976389",
        "domain": "numina",
        "question": "14th ASU 1980 Problem 17 S is a set of integers. Its smallest element is 1 and its largest element is 100. Every element of S except 1 is the sum of two distinct members of the set or double a member of the set. What is the smallest possible number of integers in S?",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 149499,
        "difficulty": null
      },
      {
        "id": "b92942a9a24f7631e0d0adf19530590c632b98f053425e988647195ed33c202f",
        "domain": "numina",
        "question": "11. If one side of the square $A B C D$ lies on the line $y=2 x-17$, and the other two vertices are on the parabola $y=x^{2}$, then the minimum value of the area of the square is $\\qquad$ .",
        "answer": "80",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 138039,
        "difficulty": null
      },
      {
        "id": "b45b5344450c0c6d9e76d4a790fdcd11efdc9c35a81da22e8145e48f78a1443c",
        "domain": "numina",
        "question": "A1 Determine the positive integer $n$ such that $8^{4}=4^{n}$.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 111916,
        "difficulty": null
      },
      {
        "id": "360b053b6acd0fed057c48c6e9eacc9f947e1cfea3d9fbcff63a49de8d61188c",
        "domain": "numina",
        "question": "Problem 8.1. In a $5 \\times 5$ square, some cells have been painted black as shown in the figure. Consider all possible squares whose sides lie along the grid lines. In how many of them is the number of black and white cells the same?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d43b1f0a7bd77fffa87ag-24.jpg?height=311&width=321&top_left_y=91&top_left_x=566)",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17407,
        "difficulty": null
      },
      {
        "id": "c6e3727e2781ac220f9349d4d85b2817da590ee5a46aaa0eb55b55c5a48fde47",
        "domain": "numina",
        "question": "4. Let's call a natural number $n$ squareable if the numbers from 1 to $n$ can be arranged in such an order that each member of the sequence, when added to its position, results in a perfect square. For example, the number 5 is squareable, as the numbers can be arranged as: 32154, in which $3+1=2+2=1+3=4$ and $5+4=4+5=9$. Determine which of the numbers $7,9,11,15$ are squareable.",
        "answer": "915",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 5851,
        "difficulty": null
      },
      {
        "id": "1f390c55f2d0bd8f02e24890b98e8ebe099a4ddd5200368740a669ada7f3b5d8",
        "domain": "numina",
        "question": "Problem 7.8. On a rectangular sheet of paper, a picture in the shape of a \"cross\" was drawn from two rectangles $A B C D$ and $E F G H$, the sides of which are parallel to the edges of the sheet. It is known that $A B=9, B C=5, E F=3, F G=10$. Find the area of the quadrilateral $A F C H$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2fdcf97aa9799d0d4cd6g-23.jpg?height=422&width=507&top_left_y=92&top_left_x=469)",
        "answer": "52.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15480,
        "difficulty": null
      },
      {
        "id": "6bc5849038667973384d580e7442872be133537b1c034a66290ed544e7eebaf4",
        "domain": "numina",
        "question": "10. Given a fixed circle $\\odot P$ with radius 1, the distance from the center $P$ to a fixed line $l$ is 2, $Q$ is a moving point on $l$, $\\odot Q$ is externally tangent to $\\odot P$, $\\odot Q$ intersects $l$ at points $M$ and $N$, for any diameter $MN$, there is a fixed point $A$ in the plane such that $\\angle MAN$ is a constant, find the degree measure of $\\angle MAN$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 98625,
        "difficulty": null
      },
      {
        "id": "dd21903a9da3d6f41667ca224abbf984692fcb02b354d704b8715c3d5a178d96",
        "domain": "numina",
        "question": "IMO 1976 Problem A1 A plane convex quadrilateral has area 32, and the sum of two opposite sides and a diagonal is 16. Determine all possible lengths for the other diagonal.",
        "answer": "8\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 149188,
        "difficulty": null
      },
      {
        "id": "9a30a5b69ba1e9560f7797b23fa5ccc9939e3f6e332c7f56e553e59175a19be7",
        "domain": "numina",
        "question": "Problem 7.8. On a rectangular sheet of paper, a picture in the shape of a \"cross\" was drawn from two rectangles $A B C D$ and $E F G H$, the sides of which are parallel to the edges of the sheet. It is known that $A B=9, B C=5, E F=3, F G=10$. Find the area of the quadrilateral $A F C H$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-23.jpg?height=422&width=507&top_left_y=92&top_left_x=469)",
        "answer": "52.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15964,
        "difficulty": null
      },
      {
        "id": "79fb4d9407f7f109fe5040a68d288dfe8e6e18b011250cdb11be2d211f34b575",
        "domain": "numina",
        "question": "9. A large rectangle is divided into four identical smaller rectangles by slicing parallel to one of its sides. The perimeter of the large rectangle is 18 metres more than the perimeter of each of the smaller rectangles. The area of the large rectangle is $18 \\mathrm{~m}^{2}$ more than the area of each of the smaller rectangles. What is the perimeter in metres of the large rectangle?",
        "answer": "28",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156417,
        "difficulty": null
      },
      {
        "id": "c68c64e14387835a2186b227bae12ee7e923ef13e40200a8a72eafe7af1892ef",
        "domain": "numina",
        "question": "4. The integer $n$ satisfies the inequality $n+(n+1)+(n+2)+\\cdots+(n+20)>2019$. What is the minimum possible value of $n$ ?",
        "answer": "87",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 156492,
        "difficulty": null
      },
      {
        "id": "94cd66c0b605f3ca9bf1b06c5179b4a59262427f5fb07f9f6bdaa0ded4671b44",
        "domain": "numina",
        "question": "12 A space station consists of 25 chambers, and each two chambers are connected by a tunnel. There are a total of 50 main tunnels that can be used in both directions, the rest are all one-way tunnels. A group of four chambers is called connected if you can get from any of these chambers to any other by using only the six tunnels that connect these chambers to each other. Determine the largest possible number of connected groups of four.\n\n## Solution",
        "answer": "9650",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 91278,
        "difficulty": null
      },
      {
        "id": "1da6780e14366ca92ed734aeed72bdc25f9d5a6003e5968717eb8a70cac5d78f",
        "domain": "numina",
        "question": "4. For a positive integer $n$, if $(x y-5 x+3 y-15)^{n}$ is expanded and like terms are combined, $x^{i} y^{j}(i, j=0,1$, $\\cdots, n)$ after combining have at least 2021 terms, then the minimum value of $n$ is $\\qquad$.",
        "answer": "44",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 106229,
        "difficulty": null
      },
      {
        "id": "01ab94e57129c23725d26c4c18147fea83f83a8f1fa6de19e673da4a36fd897c",
        "domain": "numina",
        "question": "3. A triangle $ABC$ with side lengths $AB=7 \\text{~cm}, BC=6 \\text{~cm}, CA=5 \\text{~cm}$ has been cut out of paper. It was folded along a line so that vertex $C$ ended up at point $C_{1}$ on side $AB$. Moreover, in the resulting quadrilateral $AKMB$, two angles adjacent to the fold line $KM$ turned out to be equal. Find $AC_{1}$ and $C_{1}B$.",
        "answer": "\\frac{42}{11}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 4073,
        "difficulty": null
      },
      {
        "id": "071858566c1d1a3af4958f017584fc60a366f198391b02643c9195aab882ab25",
        "domain": "numina",
        "question": "Let $P$ be a polynomial with integer coefficients such that: $P(-1)=-4, P(-3)=-40$ and $P(-5)=$\n$-156$.\n\nWhat is the maximum number of integers $x$ for which: $P(P(x))=x^{2}$?",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 84710,
        "difficulty": null
      },
      {
        "id": "609d10245cfb36db7ed360ca832aaa269979bf941e61531f0cf19e3fb3d98db9",
        "domain": "numina",
        "question": "Example 1 Write three integers on a blackboard, then erase one of them and replace it with the difference of 1 and the sum of the other two numbers. After repeating this process several times, the result is $(17,1999,2105)$. Could the three numbers initially written on the blackboard have been $(2,2,2)$ or $(3,3,3)$?",
        "answer": "132",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 106957,
        "difficulty": null
      },
      {
        "id": "0e61082310a242cb08b9dfdc111a86deb657637ec7f1cc5bf5160c6c3b494ff3",
        "domain": "numina",
        "question": "2. A rectangle with sides $6 \\mathrm{~cm}$ and $3 \\mathrm{~cm}$ was cut into three rectangles of equal perimeter. What can the perimeter of these rectangles be? Find all possible answers.",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14656,
        "difficulty": null
      },
      {
        "id": "07c8eab6c2fd79f2d2086b832b61fff77abe252b01b1d1997cd508136c1f35cc",
        "domain": "numina",
        "question": "8. [7] Triangle $A B C$ has side lengths $A B=231, B C=160$, and $A C=281$. Point $D$ is constructed on the opposite side of line $A C$ as point $B$ such that $A D=178$ and $C D=153$. Compute the distance from $B$ to the midpoint of segment $A D$.",
        "answer": "208",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151116,
        "difficulty": null
      },
      {
        "id": "bae576ed9ce0ec9fa882dc23e507b747ba74d29f96fcac211b99946d654ece08",
        "domain": "numina",
        "question": "In triangle $ABC$, the sides are known: $AB=6$, $BC=4$, $AC=8$. The bisector of angle $C$ intersects side $AB$ at point $D$. A circle is drawn through points $A$, $D$, and $C$, intersecting side $BC$ at point $E$. Find the area of triangle $ADE$.\n\n#",
        "answer": "\\frac{3\\sqrt{15}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67015,
        "difficulty": null
      },
      {
        "id": "a3bcefd4a8c4f94d72745cfb90fdacb6d35e4c89cbfbc7558433cc422634e80f",
        "domain": "numina",
        "question": "9. In how many ways can the cells of a $4 \\times 4$ table be filled in with the digits $1,2, \\ldots, 9$ so that each of the 4 -digit numbers formed by the columns is divisible by each of the 4-digit numbers formed by the rows?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 152320,
        "difficulty": null
      },
      {
        "id": "352604fc27a801d8d6d6156adfff3b54e1d688e72f0335e2fa40d74bf0972e78",
        "domain": "numina",
        "question": "6. As shown in the figure, from a rectangular cardboard with a length of 13 cm and a width of 9 cm, squares with a side length of 2 cm are cut from each corner, and then, the cardboard is folded along the dotted lines to form a rectangular container. What is the volume of this container in cubic centimeters?",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 140173,
        "difficulty": null
      },
      {
        "id": "a6ac472e3feb701ef5221c41ebea91b785a76479757162419482a3657a6711f3",
        "domain": "numina",
        "question": "24. The League of Red Death. During a raid on the headquarters of a secret organization, the police found a scrap of paper, depicted in the image.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_56f37ca27ac3d928838dg-015.jpg?height=255&width=494&top_left_y=1957&top_left_x=787)\n\n- \"I've been working on this piece of paper for three days now,\" the detective said. \"It shows the total amount of membership fees for this year: $3007.37, but the number of members (I know there are no more than 500) and the amount of one fee are obscured so that they cannot be read. How many members are there in the League of Red Death, and what is the amount of the membership fee?\"\n\nOf course, the fee cannot contain fractional cents.",
        "answer": "311",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 40940,
        "difficulty": null
      },
      {
        "id": "4881a6e5d8fe716dfe8c3b785f57361d69a31c21ec24072bce84a3d926ce6c0a",
        "domain": "numina",
        "question": "Task 1. From 3 deciliters of cough syrup, which contained $10 \\%$ distilled water, the pharmacist prepared a syrup for a child, which was supposed to contain $40 \\%$ water. How many deciliters of water did she add to the three deciliters of syrup?",
        "answer": "1.5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188624,
        "difficulty": null
      },
      {
        "id": "fb3a362e5067a7591fde6b6318b5e9fe5eda003fcd3b7611a4477bade5f373ff",
        "domain": "numina",
        "question": "10. (6 points) On her birthday, Grandma told Xiaoming: \"I have only had 18 birthdays since I was born.\" Grandma should be   years old this year.",
        "answer": "72",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 105215,
        "difficulty": null
      },
      {
        "id": "edbd59297e648675a13773343e502caccef47a01a0aad9f7e37d9a39508340c7",
        "domain": "numina",
        "question": "18. Distribution of Money. Nine friends $A, B, C, D, E, F, G$, $H, K$, gathering one evening to spend time together, did the following. First, $A$ gave each of the other eight people as much money as they already had. Then $B, C$, and so on up to $K$ did the same. After this, it turned out that all nine friends had the same amount of money.\n\nCan you find the smallest sum in cents that each of the participants could have had initially?",
        "answer": "512",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 40934,
        "difficulty": null
      },
      {
        "id": "bf111f4982f4df7537b1904fb4b2be4723115ff0dbe593f9e5f982c68f250666",
        "domain": "numina",
        "question": "53. Find a three-digit number, every integer power of which ends in the three digits that make up the original number.",
        "answer": "625376",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 29325,
        "difficulty": null
      },
      {
        "id": "bae93c6350ab14f5a65d536c35c496e57444598810f7e38105f42455a60f3df1",
        "domain": "numina",
        "question": "Write in ascending order the multiples of 3 that, when added to 1, are perfect squares, that is, $3,15,24,48, \\ldots$ What is the multiple of 3 in the $2006^{\\mathrm{th}}$ position?",
        "answer": "9060099",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 74700,
        "difficulty": null
      },
      {
        "id": "95004a328906406d1dc84240bfd88ea66103328121bcf1eddd2a5e4b9b427ffc",
        "domain": "numina",
        "question": "2. Joško added the largest three-digit number divisible by 9 and the smallest three-digit number not divisible by 9, while Fran added the largest three-digit number not divisible by 9 and the smallest three-digit number divisible by 9. Which sum is greater and by how much?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86850,
        "difficulty": null
      },
      {
        "id": "44abc427958128189fb6e6d342d8e6e00b1a0cffceeae3487e3cb91c8d5443a9",
        "domain": "numina",
        "question": "In the city of Oropis, there is a triangular lake with each of its three sides being part of the perimeter of a square-shaped land with areas of $370 \\mathrm{~m}^{2}$, $116 \\mathrm{~m}^{2}$, and $74 \\mathrm{~m}^{2}$, as shown in the first figure below. The mayor of Oropis, Arnaldo, wants to calculate the area of the lake but doesn't know how. The mayor's assistant, Bernaldo, has an idea. He says that it is enough to find integer values of $a, b, c$, and $d$ that satisfy the geometric conditions of the second figure, where the shaded area is congruent to the lake. He claims that after this, the task will become much simpler.\n![](https://cdn.mathpix.com/cropped/2024_05_01_10107d2fba0ec26bc845g-33.jpg?height=892&width=1442&top_left_y=766&top_left_x=404)\n\n(a) Determine integer values of $a, b, c$, and $d$ that satisfy the geometric conditions of Bernaldo's figure.\n\n(b) Determine the area of the lake.",
        "answer": "11",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74569,
        "difficulty": null
      },
      {
        "id": "18e2c6c7599c9fe65183695b3fb2934d70ce282666b7cc14b1f1c59be9e9de92",
        "domain": "numina",
        "question": "7. If the set $S=\\{1,2, \\cdots, n\\}$ has three-element subsets $A_{1}, A_{2}, \\cdots, A_{m}(m>1)$ satisfying:\n(1) Each two-element subset of $S$ is contained in exactly one $A_{i}$ $(1 \\leqslant i \\leqslant m)$;\n(2) $A_{i}$ and $A_{j}(i \\neq j, 1 \\leqslant i, j \\leqslant m)$ have exactly one common element.\nThen $m=$ . $\\qquad$",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 133346,
        "difficulty": null
      },
      {
        "id": "dc043b01bdcd6284754395e0070e52854153430a0c316aeff3bb9ea9d422b20e",
        "domain": "numina",
        "question": "3. The weight is made of an alloy of four metals and weighs 16.3 kg. The weight of the first metal in this alloy is one and a half times greater than that of the second, the weight of the second metal is to the weight of the third as $3: 4$, and the weight of the third to the weight of the fourth - as $5: 6$. Determine the weight of the fourth metal.",
        "answer": "4.8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11402,
        "difficulty": null
      },
      {
        "id": "7f6bb0353795eca568a94f7f9676a0d9a6a816b32f9007b0d25d95238dc4314d",
        "domain": "numina",
        "question": "22. Yura left the house for school 5 minutes later than Lena, but walked twice as fast as she did. How many minutes after leaving will Yura catch up to Lena?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39631,
        "difficulty": null
      },
      {
        "id": "19efbef9a035aca605b0ba595b19a02f9d840225ee0c2d34c5084ebbd89f1f5d",
        "domain": "numina",
        "question": "12. (12 points) The figure below is a $6 \\times 6$ grid. Fill in the blank cells with the numbers $1 \\sim 6$ such that each row and each column contains the numbers $1 \\sim 6$ exactly once. The grid is also divided into 6 regions by bold lines, and each region must contain the numbers $1 \\sim 6$ exactly once. What is the four-digit number $\\overline{\\mathrm{ABCD}}$ formed by the first 4 numbers in the bottom row? $\\qquad$",
        "answer": "2413",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 118687,
        "difficulty": null
      },
      {
        "id": "13b617819551a27298c250d5f50be646bd7db35fa28dd9cd6585461a003bd102",
        "domain": "numina",
        "question": "Right-angled triangle $ABC$ has legs $|AB|=5 \\text{ cm}$ and $|BC|=12 \\text{ cm}$. The vertices of triangle $ABC$ are mapped such that\n\n- point $A$ is mapped to point $A^{\\prime}$ symmetrically with respect to the angle bisector of $\\angle ABC$,\n- point $B$ is mapped to point $B^{\\prime}$ symmetrically with respect to the midpoint of segment $BC$,\n- point $C$ is mapped to point $C^{\\prime}$ symmetrically with respect to the perpendicular bisector of segment $AB$.\n\nDetermine the area of triangle $A^{\\prime} B^{\\prime} C^{\\prime}$.\n\n(M. Dillingerová)",
        "answer": "17.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76402,
        "difficulty": null
      },
      {
        "id": "d026d1ea081674e44e4d21bb706ad0973550b040b6df4a2402239f41e30e3d46",
        "domain": "numina",
        "question": "$4 \\cdot 202$ City A has 157 tons of goods to be transported to City B, large trucks have a load capacity of 5 tons, small trucks have a load capacity of 2 tons, large trucks consume 20 liters of fuel, small trucks consume 10 liters of fuel, how many large trucks and small trucks should be used to minimize fuel consumption?",
        "answer": "31",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115124,
        "difficulty": null
      },
      {
        "id": "d10937f56b30b07b9718162d57d108cbbf7f1e3b4de5cfc3b8819880cdca64f2",
        "domain": "numina",
        "question": "4. If the numerator of a fraction is reduced by $8\\%$, and the denominator is increased by $8\\%$, the new fraction will be 2 less than the original fraction. Determine the value of the original fraction.",
        "answer": "\\frac{27}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86659,
        "difficulty": null
      },
      {
        "id": "1402d30157fd0bfff8173dc31bdb77b23a6f11c9ca41d92f907b6fb42d80127d",
        "domain": "numina",
        "question": "20. (3 points) The PLA soldiers need to repair a dam while the flood continuously destroys it. If 10 people need 45 minutes, and 20 people need 20 minutes, then 14 people need $\\qquad$ minutes to repair the dam.",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131047,
        "difficulty": null
      },
      {
        "id": "f1410b7baee5e17c59853465ad2df53b967665035a299da34a6b528bb762f527",
        "domain": "numina",
        "question": "Problem 10.6. In a convex quadrilateral $A B C D$, the midpoint of side $A D$ is marked as point $M$. Segments $B M$ and $A C$ intersect at point $O$. It is known that $\\angle A B M=55^{\\circ}, \\angle A M B=$ $70^{\\circ}, \\angle B O C=80^{\\circ}, \\angle A D C=60^{\\circ}$. How many degrees does the angle $B C A$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-39.jpg?height=371&width=709&top_left_y=89&top_left_x=372)",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13452,
        "difficulty": null
      },
      {
        "id": "bd2606ea45cdeda2f76534ed2328ecdaeb5341548f3bf1542e7eaba55c5b6c8c",
        "domain": "numina",
        "question": "3. (8 points) In rectangle $A B C D$, $B E=5, E C=4, C F=4, F D=1$, as shown in the figure, then the area of $\\triangle A E F$ is $\\qquad$ ;",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 100359,
        "difficulty": null
      },
      {
        "id": "9e76ca09c40d6ac169344d3abccba4160ad8886b8a7810f558832e2c71e4b263",
        "domain": "numina",
        "question": "Kirienko d:\n\nSasha and Masha each thought of a natural number and told them to Vasya. Vasya wrote the sum of the numbers on one piece of paper and their product on another, then hid one of the papers and showed the other (which had the number 2002 written on it) to Sasha and Masha. Seeing this number, Sasha said that he did not know what number Masha had thought of. Hearing this, Masha said that she did not know what number Sasha had thought of. What number did Masha think of?",
        "answer": "1001",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 70307,
        "difficulty": null
      },
      {
        "id": "4369e5344d06ac22304bca5fc9b622f6abe7e14146fe2d54ae2e9d8d3dc9c244",
        "domain": "numina",
        "question": "Ggrmanova T.A.\n\nTanya sequentially wrote down numbers of the form $\\{n \\wedge 7-1\\}$ for natural numbers $n=2,3, \\dots$ and noticed that for $n=8$ the resulting number is divisible by 337. For what smallest $n>1$ will she get a number divisible by $2022?$",
        "answer": "79",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 68581,
        "difficulty": null
      },
      {
        "id": "827b0d4cfb7575752ab30f877f5f104a8f4c09dff5108c21789d74d9e20b7a95",
        "domain": "numina",
        "question": "In triangle $A B C$, $A B=A C$ and $\\angle B A C=20^{\\circ}$. Let $E$ and $F$ be the points on sides $A B$ and $A C$ such that $\\angle B C E=50^{\\circ}$ and $\\angle C B F=60^{\\circ}$. What angle do the lines $B C$ and $E F$ form? (H)",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 166556,
        "difficulty": null
      },
      {
        "id": "714d3b2afda8e18e55e7084d825ef938ccef901876846591268ad8b2e2197b11",
        "domain": "numina",
        "question": "6. For $n \\in \\mathbf{N}^{*}$, let $S_{n}$ be the minimum value of $\\sum_{k=1}^{n} \\sqrt{(2 k-1)^{2}+a_{k}^{2}}$, where $a_{1}, a_{2}, \\cdots, a_{n} \\in \\mathbf{R}^{+}$, and $\\sum_{k=1}^{n} a_{k}=17$. If there exists $n$ such that $S_{n}$ is also an integer, find all values of $n$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134007,
        "difficulty": null
      },
      {
        "id": "9510c3bf5d46f2671bfd8ae5cc84037f38d3093cd85a14e3829bb4de5de5d935",
        "domain": "numina",
        "question": "3. In the diagram, $O A B$ is a quarter circle, $O E$ and $C D$ are parallel, $B C$ and $E A$ are parallel, and $\\angle B C D=$ $4 \\times \\angle O B C$.\nWhat is the size of $\\angle O B C$ ?\n\nQuadrilateral $F C G E$ must be a parallelogram, as $O E$ is parallel to $C D$ and $B C$ is parallel to $E A$ (given in the question).\n\nWe have that $\\angle B C D=\\angle A E O=4 x^{\\circ}$, as opposite angles in a parallelogram are equal and we are told in the question that $\\angle B C D=4 \\times \\angle O B C$.\n\nTriangle $O A E$ is isosceles as $O A=O E$ because both are radii of the quarter circle $O A B$, thus $\\angle O A E=\\angle A E O=$ $4 x^{\\circ}$.\n\nAlso $\\angle O C B=\\angle O A E$, as $B C$ is parallel to $E A$ and hence they are corresponding angles.\n\nAs $O A B$ is a quarter circle then $\\angle B O A=90^{\\circ}$.\nConsidering the sum of the interior angles of the right-angled triangle $O C B$,\n$$\n180^{\\circ}=90^{\\circ}+4 x^{\\circ}+x^{\\circ}\n$$\n\nFinally, the previous equation simplifies to $90^{\\circ}=5 x^{\\circ}$ and so $\\angle O B C=x^{\\circ}=18^{\\circ}$.",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 154967,
        "difficulty": null
      },
      {
        "id": "822c1ef077b5fcda7c225727332e84314008e49f36fffc7ba82b83d9cb4dbe73",
        "domain": "numina",
        "question": "B3. The diagram shows the distribution of bronze, silver, and gold medals at a competition.\n\n(a) What is the value of $x$? (see image)\n\n(b) Write the ratio of gold, silver, and bronze medals in the smallest possible natural numbers.\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_b93fa657b92bc897cb7bg-05.jpg?height=417&width=423&top_left_y=126&top_left_x=1496)\n\n(c) At the competition, 84 medals were awarded. How many bronze, how many silver, and how many gold medals were awarded?\n\n(d) How many grams does a bronze medal weigh if it has the shape of a cylinder with a diameter of $7 \\mathrm{~cm}$ and a height of 3 $\\mathrm{mm}$? The density of bronze is $8700 \\frac{\\mathrm{kg}}{\\mathrm{m}^{3}}$.\n\n## 21st Mathematics Knowledge Competition for Vocational School Students National Competition, May 15, 2021\n\n## Problems for 3rd Year\n\nTime for solving: 90 minutes. In section A, we will award two points for a correct answer, while we will deduct half a point for an incorrect answer. Enter the answers for section A in the left table. In section B, we will award up to seven points for a correct answer.\n![](https://cdn.mathpix.com/cropped/2024_06_07_b93fa657b92bc897cb7bg-06.jpg?height=162&width=1198&top_left_y=590&top_left_x=444)",
        "answer": "100",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 163414,
        "difficulty": null
      },
      {
        "id": "b3ff45aa203a632b6f95df0c65b02340c997c5505a4d5cf8972e7d4034cd7c3a",
        "domain": "numina",
        "question": "7.208. $2^{x-1}+2^{x-4}+2^{x-2}=6.5+3.25+1.625+\\ldots$ (the expression on the right side is an infinite geometric progression).",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 48383,
        "difficulty": null
      },
      {
        "id": "b99d1022587ee4cabbeb94a6efd427d96253fe9622c5e11f83dedd29c6843a2b",
        "domain": "numina",
        "question": "9. As shown in the figure, $A B C D$ and $E G F O$ are both squares, where point $O$ is the center of square $A B C D$, and $E F / / B C$. If the lengths of $B C$ and $E F$ are both positive integers, and the area of quadrilateral $B C E F$ is 3.25, then $\\mathrm{S}_{A B C D}-\\mathrm{S}_{E G F O}=$ $\\qquad$ ($\\mathrm{S}_{E G F O}$ represents the area of $E G F O$, and so on).",
        "answer": "11.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124379,
        "difficulty": null
      },
      {
        "id": "c0ed9476c921280dec0827f823da8d16be45ced807752a8475b45cd13116be13",
        "domain": "numina",
        "question": "6.5. The arithmetic mean of four numbers is 10. If one of these numbers is erased, the arithmetic mean of the remaining three increases by 1; if instead another number is erased, the arithmetic mean of the remaining numbers increases by 2; and if only the third number is erased, the arithmetic mean of the remaining increases by 3. By how much will the arithmetic mean of the remaining three numbers change if the fourth number is erased?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17957,
        "difficulty": null
      },
      {
        "id": "047bcc2eb000fef414be3fa7c653b48570e5d00d1fa84012ab7e73b73a5be57a",
        "domain": "numina",
        "question": "52. Given a triangle $T$.\n\na) Place a centrally symmetric polygon $m$ of the largest possible area inside $T$.\n\nWhat is the area of $m$ if the area of $T$ is 1?\n\nb) Enclose $T$ in a convex centrally symmetric polygon $M$ of the smallest possible area.\n\nWhat is the area of $M$ if the area of $T$ is 1?",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 25149,
        "difficulty": null
      },
      {
        "id": "1ccb96d5fb779b411499d088910e1b504ef908b238e6905e37f9ff1032ecb5f9",
        "domain": "numina",
        "question": "Task B-1.6. Solve the equation\n\n$$\n\\frac{x+3}{12(x+1)}:\\left(\\frac{2 x-3}{3 x-3}-\\frac{3 x-1}{4 x+4}+\\frac{x^{2}-7 x+14}{12 x^{2}-12}\\right)=2015\n$$",
        "answer": "2012",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87334,
        "difficulty": null
      },
      {
        "id": "f5e029dba4b3ac61d895dac10b797dffaa41dfb5e16579c0b186792656353467",
        "domain": "numina",
        "question": "9. On the table, there are param 1 externally identical coins. It is known that among them, there are exactly param 2 counterfeit ones. You are allowed to point to any two coins and ask whether it is true that both these coins are counterfeit. What is the minimum number of questions needed to guarantee getting at least one \"Yes\" answer?\n\n| param1 | param2 |  |\n| :---: | :---: | :---: |\n| 120 | 60 |  |",
        "answer": "63",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 3104,
        "difficulty": null
      },
      {
        "id": "2d012931915ac6ae363327471476c5515b84f5cb34c358938ecaea5d8d28f997",
        "domain": "numina",
        "question": "A1 Determine the positive integer $n$ that satisfies the following equation:\n$$\n\\frac{1}{2^{10}}+\\frac{1}{2^{9}}+\\frac{1}{2^{8}}=\\frac{n}{2^{10}} .\n$$",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108851,
        "difficulty": null
      },
      {
        "id": "989bea2fdc578421cc3e4ccb539cf56e966198a49d51b34f67ec37f5171a1f6a",
        "domain": "numina",
        "question": "74. A school uses a drug fumigation method to disinfect classrooms. It is known that when the drug is burning, the content of the drug $y(\\mathrm{mg})$ per cubic meter of air is directly proportional to the time $x$ (min); after the drug burns out, $y$ is inversely proportional to $x$ (as shown in the figure). It is measured that the drug burns out in $8 \\mathrm{~min}$, at which point the content of the drug in the air per cubic meter is $6 \\mathrm{mg}$. Please answer the following questions based on the information provided in the question:\n(1) When the drug is burning, the function relationship of $y$ with respect to $x$ is: $\\qquad$, the range of the independent variable $x$ is: $\\qquad$, after the drug burns out, the function relationship of $y$ with respect to $x$ is: $\\qquad$;\n(2) Research indicates that students can return to the classroom only when the content of the drug in the air per cubic meter is below $1.6 \\mathrm{mg}$. How many minutes at least from the start of disinfection must pass before students can return to the classroom? $\\qquad$ $\\min$;\n(3) Research indicates that the air-borne pathogens can be effectively killed only when the content of the drug in the air per cubic meter is no less than $3 \\mathrm{mg}$ and lasts for no less than $10 \\mathrm{~min}$. Is this disinfection effective? Answer: $\\qquad$",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93449,
        "difficulty": null
      },
      {
        "id": "e1af691b8229865ec181058cdeb84f8c2be41f32d4cca21463ae0442f9c34fba",
        "domain": "numina",
        "question": "How many rectangles can be seen in the figure, and how many of them are squares? (There are only two different distances between adjacent parallel lines, their ratio is $1: 2$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_02_f7d7d7aa1e130da3e5a8g-1.jpg?height=696&width=702&top_left_y=270&top_left_x=703)",
        "answer": "6975",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 164495,
        "difficulty": null
      },
      {
        "id": "1afd19adcc8082d42d8a2814122d727184aa872e46c543bf73157a0b1e31d07e",
        "domain": "numina",
        "question": "11. (5 points) Arrange the natural numbers $1,2,3,4$, in ascending order without any gaps, to get: 1234567891011121314. In this sequence of digits, when the first set of 5 consecutive even digits appears, the position of the first (even) digit from the left is the how many-th digit?",
        "answer": "490",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105024,
        "difficulty": null
      },
      {
        "id": "6eae107c32a74441a6560cd904bad039091c28aea4e64991d1226bcc2f238fcf",
        "domain": "numina",
        "question": "4. (10 points) The product of three consecutive natural numbers is 120, their sum is $\\qquad$",
        "answer": "15",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92738,
        "difficulty": null
      },
      {
        "id": "1a09e2e0beeb428b2fd78ca85c1f7ff53a2df0f6c828d05ab55bc8a33fe376c5",
        "domain": "numina",
        "question": "4. (5 points) Definition: If $a: b=b: c$, then $b$ is called the mean proportional between $a$ and $c$; for example, $1: 2=2: 4$, so 2 is the mean proportional between 1 and 4. $\\qquad$",
        "answer": "0.48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120080,
        "difficulty": null
      },
      {
        "id": "7c55ac24e71f85093d4ef718afe6c9936fcf4d875799650b5ff2231651e6da0a",
        "domain": "numina",
        "question": "8. (10 points) The integer $n$ has a total of 10 factors. These factors, arranged in ascending order, have the 8th one as $\\frac{\\mathrm{n}}{3}$. What is the maximum value of the integer $n$? $\\qquad$ .",
        "answer": "162",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94080,
        "difficulty": null
      },
      {
        "id": "98dc8b576bc56b4a69337aae2565dfb3f4bc930f85e4ce89bcccb29b1aaff271",
        "domain": "numina",
        "question": "Problem 10.1. Find the least natural number $a$ such that the equation $\\cos ^{2} \\pi(a-x)-2 \\cos \\pi(a-x)+\\cos \\frac{3 \\pi x}{2 a} \\cdot \\cos \\left(\\frac{\\pi x}{2 a}+\\frac{\\pi}{3}\\right)+2=0$ has a root.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 177362,
        "difficulty": null
      },
      {
        "id": "1e435a9bedfa4e5304a419778fe303159115240c875cc1d58cd8712b9527986e",
        "domain": "numina",
        "question": "5. A square $A B C D$ has side-length 2 , and $M$ is the midpoint of $B C$. The circle $S$ inside the quadrilateral $A M C D$ touches the three sides $A M, C D$ and $D A$. What is its radius?",
        "answer": "3-\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 155285,
        "difficulty": null
      },
      {
        "id": "1477e690f302cb531f03d10abc11cf603c4e922517feacc0cfcf140d11090842",
        "domain": "numina",
        "question": "1.3. Replace the ellipsis with such a natural number $p$ so that the answer to the following question is unique: how many lines are drawn on the plane, if it is known that they intersect at ... different points?",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 25642,
        "difficulty": null
      },
      {
        "id": "68562804a392c470f19d74ceb12e7df20c6ae7c37b2dcfd8f51e328e00f8d73f",
        "domain": "numina",
        "question": "1. In a recipe for soup, it says that for 5 servings of this soup, 30 dag of carrots are needed. If the cook wants to cook soup for 60 people and prepare 2 servings of soup per person, how many carrots should be put into this soup?",
        "answer": "720",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86763,
        "difficulty": null
      },
      {
        "id": "a2991f170d0abb47d6371b1f8e89043c70abfeff2fb181fc1af93fb9a6e6c1d8",
        "domain": "numina",
        "question": "5. Consider a $2008 \\times 2008$ chess board. Let $M$ be the smallest number of rectangles that can be drawn on the chess board so that the sides of every cell of the board is contained in the sides of one of the rectangles. Find the value of $M$. (For example, for the $2 \\times 3$ chess board, the value of $M$ is 3 .)",
        "answer": "2009",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160331,
        "difficulty": null
      },
      {
        "id": "120f1f46e8e210773733cac4da33be82c3d91555b4d1cffa0d1681eefe2d6292",
        "domain": "numina",
        "question": "A positive integer that is the same when read forwards or backwards is called a palindrome. A table is created, as shown, in which the first column contains all palindromes between 1000 and 10000 , listed in\n\n| Palindrome | Difference |\n| :---: | :---: |\n| 1001 | 110 |\n| 1111 | 110 |\n| 1221 | 110 |\n| 1331 | 110 |\n| 1441 | $\\vdots$ |\n| $\\vdots$ |  |\n\nporitive diference between each pair of consecutive palindromes. For example, 1001 and 1111 are consecutive palindromes and their positive difference is $1111-1001=110$.\n\n(a) What are the eighth and ninth palindromes in the first column of the table?\n\n(b) The second column of the table contains only two different numbers: 110 and $x$. Determine the value of $x$.\n\n(c) When the table is complete, there are $N$ palindromes in the first column. What is the value of $N$ ?\n\n(d) Determine the average of all of the $N-1$ numbers in the second column of the table.",
        "answer": "\\frac{8998}{89}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 193956,
        "difficulty": null
      },
      {
        "id": "feff4100043c688caf518ca2205446876022964c5cf22505350af0a6295128dc",
        "domain": "numina",
        "question": "9.8 The commission consists of a chairman, his deputy, and five other people. In how many ways can the commission members distribute among themselves the duties of chairman and deputy?",
        "answer": "42",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 49865,
        "difficulty": null
      },
      {
        "id": "740e143d28960b5c018b19844c50fb454133195f0a4fcdc019dde8b5d88144e6",
        "domain": "numina",
        "question": "The 25th question: In country A with 2011 cities, there is a unique bidirectional flight route operated by a certain airline between any city in A and any city in country B, which also has 2011 cities. There are no flight routes between cities within the same country, and at most 19 different airlines operate these routes. Find the maximum integer $\\mathrm{k}$, such that no matter how the routes are arranged, there are always $\\mathrm{k}$ cities that are connected (one can travel from one city to another, not necessarily directly), and all the routes are operated by a fixed airline.",
        "answer": "212",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 129756,
        "difficulty": null
      },
      {
        "id": "ed112f6a03ad7beca493a0eb03ef3327447db62b1e13d79c7705a741f44dcee2",
        "domain": "numina",
        "question": "4. Let $x$ and $y$ be positive numbers whose sum is 2. Find the maximum value of the expression $x^{2} y^{2}\\left(x^{2}+y^{2}\\right)$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 43275,
        "difficulty": null
      },
      {
        "id": "b15adb243b134dbe673f2144484096d4c97d3212238096535fe18ef01855a08b",
        "domain": "numina",
        "question": "1.3.7 ** Let the quadratic function $f(x)=a x^{2}+b x+c(a, b, c \\in \\mathbf{R}, a \\neq 0)$ satisfy the following conditions:\n（1）For $x \\in \\mathbf{R}$, $f(x-4)=f(2-x)$, and $f(x) \\geqslant x$;\n（2）For $x \\in(0,2)$, $f(x) \\leqslant\\left(\\frac{x+1}{2}\\right)^{2}$;\n(3) The minimum value of $f(x)$ on $\\mathbf{R}$ is 0.\n\nFind the largest $m(m>1)$ such that there exists $t \\in \\mathbf{R}$, for any $x \\in[1, m]$, we have $f(x+t) \\leqslant x$.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109515,
        "difficulty": null
      },
      {
        "id": "b544633c57d38005de6c96900721a2f43eae25274ade4ac68782f4cba3a56ea2",
        "domain": "numina",
        "question": "## Task 2.\n\nOn each field of an $n \\times n$ board, a dragon is sleeping. Dragons are neighbors if they are on fields that share a side. Every hour, Mini wakes up one dragon that has at least one living neighbor, and Maks directs that dragon towards one of its living neighbors. The awakened dragon breathes fire and destroys the chosen neighbor, then goes back to sleep.\n\nMini wants to reduce the number of dragons, while Maks wants as many dragons to remain as possible. If both choose the best they can, how many dragons will remain on the board for\n\n(a) $n=4$ ?\n\n(b) $n=5$ ?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89859,
        "difficulty": null
      },
      {
        "id": "bd8a3330e9538debd8fea8f8ae955b901665e3fb217e59f4ca1529179b4939ee",
        "domain": "numina",
        "question": "We write down all the numbers from 1 to one million successively. Then we replace each number with the sum of its digits. Then we repeat this process until only single-digit numbers remain. Which digit appears most frequently?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 84549,
        "difficulty": null
      },
      {
        "id": "c721cd702fe3e317596ea76c1a81430943d892c266879e5cab37bfeec983f051",
        "domain": "numina",
        "question": "The sum of two positive integers is 60 and their least common multiple is 273 . What are the two integers?\n\n(The least common multiple of two positive integers is the smallest positive integer which is a multiple of these two integers.)",
        "answer": "2139",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 193925,
        "difficulty": null
      },
      {
        "id": "f7b3642a49d7a1b3b7f5dca104c14c90be1cc615a28191db6463deea48230695",
        "domain": "numina",
        "question": "Let $a_{1}, a_{2}, \\ldots$ be a sequence of integers that has infinitely many positive and infinitely many negative elements, and for every $n$, it holds that $a_{1}, a_{2}, \\ldots, a_{n}$ leave pairwise distinct remainders when divided by $n$. How many times does $2008$ appear in the sequence?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 169700,
        "difficulty": null
      },
      {
        "id": "ec0bfce600e5ea12281d1f5eb06757c0faca26392e714a58ef29548df449c583",
        "domain": "numina",
        "question": "8. Given that the parabola $P$ has the center of the ellipse $E$ as its focus, $P$ passes through the two foci of $E$, and $P$ intersects $E$ at exactly three points, then the eccentricity of the ellipse is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{2\\sqrt{5}}{5}",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 132760,
        "difficulty": null
      },
      {
        "id": "29871b3c98f4eb64aad293dbe01a767caab1622086a8ff5ef419b27562e65bb8",
        "domain": "numina",
        "question": "G6.4 If $r, s, t, u$ are positive integers and $r^{5}=s^{4}, t^{3}=u^{2}, t-r=19$ and $d=u-s$, find the value of $d$.",
        "answer": "757",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 152793,
        "difficulty": null
      },
      {
        "id": "daf3e813af2b866e4b9fd2fabe1e5d906d78d6842aec4cbcfcae006a652f84ad",
        "domain": "numina",
        "question": "8,9 |\n\nThe diagonal of a rectangular trapezoid and its lateral side are equal.\n\nFind the midline of the trapezoid if the height of the trapezoid is 2 and the lateral side is 4.\n\n#",
        "answer": "3\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66723,
        "difficulty": null
      },
      {
        "id": "6c7075c96a303c56714394fb93a5000ffd7ac0e57b9c51e0903827b732d58f2b",
        "domain": "numina",
        "question": "80. In $\\triangle A B C$, $C P=\\frac{1}{3} B C, C Q=\\frac{1}{4} A C, B Q$ intersects $A P$ at point $N$. If the area of $\\triangle A B C$ is $12$, the area of $\\triangle A B N$ is $\\qquad$",
        "answer": "\\frac{72}{11}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 91789,
        "difficulty": null
      },
      {
        "id": "6f7443d1d953864fffc29cbabff6850b14badca7bda1d21e90b451c4022bae8f",
        "domain": "numina",
        "question": "Example 5 Two periodic functions $y_{1}$ and $y_{2}$ have the smallest positive periods of $a$ and $b$ respectively, and $b=n a$, where $n \\geqslant 2, n \\in \\mathbf{N}$. If the function $y=y_{1}+y_{2}$ has the smallest positive period of $t$, then among the following 5 scenarios: (1) $t<b$;\n(5) $a<t<b$, the possible scenarios are $\\qquad$ (fill in the sequence number).",
        "answer": "(1)(3)(5)",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104065,
        "difficulty": null
      },
      {
        "id": "545ea60a2d9cf7f0a50c48794866041be26e446f8cc3e3863cb0e3004290531e",
        "domain": "numina",
        "question": "3. In how many ways can you partition the set $\\{1,2, \\ldots, 12\\}$ into six mutually disjoint two-element sets in such a way that the two elements in any set are coprime?\n\n(Martin Panák)",
        "answer": "252",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 76981,
        "difficulty": null
      },
      {
        "id": "4a670e8343120977cae06e130ce2b8a0f6cfb1864bbacb732aa36a9b0f74acd4",
        "domain": "numina",
        "question": "9. (15 points) As shown in the figure, two squares $A B F E$ and $C D E F$ with side lengths of 6 are combined to form rectangle $A B C D$. $G$ is the midpoint of $D E$. Connect $B G$ and it intersects $E F$ at $H$. Find the area of pentagon $C D G H F$ in the figure.",
        "answer": "33",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 108208,
        "difficulty": null
      },
      {
        "id": "eb77355dae887f701b13e35e99f5be978638a53d960ea5910c0c1cd340ee01f7",
        "domain": "numina",
        "question": "6. The minor axis of the ellipse $\\rho=\\frac{1}{2-\\cos \\theta}$ is equal to\n\n保留了源文本的换行和格式。",
        "answer": "\\frac{2}{3}\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 101373,
        "difficulty": null
      },
      {
        "id": "7698a721727df943103ae32f290776d387befded55a377712b5086d76c998d64",
        "domain": "numina",
        "question": "1. Solve the equation $3 \\cdot 9^{x}+2 \\cdot 3^{x}=1$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19172,
        "difficulty": null
      },
      {
        "id": "7b7af224738bedd30593c782547af239a31876de1b6f3ae53732aca467429a64",
        "domain": "numina",
        "question": "7. Let $A B C D E$ be a square pyramid of height $\\frac{1}{2}$ with square base $A B C D$ of side length $A B=12$ (so $E$ is the vertex of the pyramid, and the foot of the altitude from $E$ to $A B C D$ is the center of square $A B C D)$. The faces $A D E$ and $C D E$ meet at an acute angle of measure $\\alpha$ (so that $0^{\\circ}<\\alpha<90^{\\circ}$ ). Find $\\tan \\alpha$.",
        "answer": "\\frac{17}{144}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151560,
        "difficulty": null
      },
      {
        "id": "4bc28b177f1d4c0cd25cf5a120a39468287c473f6091bb5aca868c17931fb30f",
        "domain": "numina",
        "question": "3.21 A motorcyclist set off from point $A$ to point $B$, which is 120 km away from $A$. On the way back, he set off at the same speed, but had to stop for 10 minutes one hour after departure. After this stop, he continued his journey to $A$, increasing his speed by 6 km/h. What was the initial speed of the motorcyclist, given that he spent as much time on the return trip as he did on the trip from $A$ to $B$?",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49561,
        "difficulty": null
      },
      {
        "id": "d7df7dd40701dc9df9617a3dbf50d442981494143460bc254e920c0c70685c15",
        "domain": "numina",
        "question": "3. The circle $(x+3)^{2}+(y-4)^{2}=50$ and the line $y=2 x-5$ intersect at two points. Determine the distance between these two points.",
        "answer": "2\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 154657,
        "difficulty": null
      },
      {
        "id": "da701795be060aab2e8629e666fcff3400ed7287fcf8f2e70f52c678eb953ce4",
        "domain": "numina",
        "question": "5. A five-digit number with all distinct digits, if the sum of any two of its digits is not 7, is called a “hidden seven number”. There are $\\qquad$ such “hidden seven numbers”.",
        "answer": "7008",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146077,
        "difficulty": null
      },
      {
        "id": "7c2832ac3439d52e41f72c4987484182d75c03ab44fc09e0653de98007633b2d",
        "domain": "numina",
        "question": "28. At the meeting, there are 25 people present. In how many ways can four people be elected from them to fill 4 different positions",
        "answer": "303600",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44849,
        "difficulty": null
      },
      {
        "id": "c4dce2c21725af75e2b2b9a789d3ec971c6df676a366661eb9dfe5d4e3a1dfb8",
        "domain": "numina",
        "question": "1. Several boys and girls gathered around a round table. It is known that exactly for 7 girls, the person sitting to their left is a girl, and for 12 - a boy. It is also known that for $75\\%$ of the boys, the person sitting to their right is a girl. How many people are sitting at the table?",
        "answer": "35",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5918,
        "difficulty": null
      },
      {
        "id": "b450d30ce343a90d06bec8c5cf028c3d9200f0135e67059f5375b81e162ad1ba",
        "domain": "numina",
        "question": "2. Let\n$$\nX=\\left\\{\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right) \\mid x_{1}, x_{2}, \\cdots, x_{n} \\in \\mathbf{Z}\\right\\}\n$$\n\nbe the set of all lattice points in $n$-dimensional space. For any two lattice points $A\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right)$ and $B\\left(y_{1}, y_{2}, \\cdots, y_{n}\\right)$ in $X$, define their distance as\n$$\n|A-B|=\\left|x_{1}-y_{1}\\right|+\\left|x_{2}-y_{2}\\right|+\\cdots+\\left|x_{n}-y_{n}\\right| .\n$$\n\nFind the minimum value of $r$ such that the points in $X$ can be $r$-colored, satisfying the following conditions:\n(1) Any two points of the same color $A$ and $B$ satisfy\n$$\n|A-B| \\geqslant 2 \\text {; }\n$$\n(2) Any two points of the same color $A$ and $B$ satisfy\n$$\n|A-B| \\geqslant 3 \\text {. }\n$$",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125377,
        "difficulty": null
      },
      {
        "id": "8345e7838f9fd790396f2fb1ee854dd5f6c746b0d5fe15fe5085d061dfb7485f",
        "domain": "numina",
        "question": "$3.457 A=\\operatorname{ctg}\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arcctg}\\left(-\\frac{1}{2}\\right)\\right)$\n\n$3.457 A=\\cot\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arcctg}\\left(-\\frac{1}{2}\\right)\\right)$",
        "answer": "-2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51888,
        "difficulty": null
      },
      {
        "id": "88bd77525b83a3556ecaf512f8e02b0b6dae1a6cb1e4837597c5ee0a874bb57f",
        "domain": "numina",
        "question": "3. Given that $\\triangle A B C$ is an equilateral triangle, the ellipse $\\Gamma$ has one focus at $A$, and the other focus $F$ lies on the line segment $B C$. If the ellipse $\\Gamma$ passes exactly through points $B$ and $C$, then its eccentricity is $\\qquad$.",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119652,
        "difficulty": null
      },
      {
        "id": "df4353ba02e3afeb3df76c2e3cf5a2353daa1a18a7e6731bde2e480db3d10f0e",
        "domain": "numina",
        "question": "A number appears on a computer screen. We know that if $x$ appears on the screen, the number $x^{2}-2 x+1$ appears right after. If the first number to appear is 2, what is the 2020-th number to appear",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 84486,
        "difficulty": null
      },
      {
        "id": "bbe9ae15082394bef60fd460239b41c599c0cf15bbbe95cdeb8be61cd6a81350",
        "domain": "numina",
        "question": "3A. Given the set of quadratic functions $f(x)=a x^{2}+b x+c$, for which $a<b$ and $f(x) \\geq 0$ for every $x \\in \\mathbb{R}$. Determine the smallest possible value of the expression $A=\\frac{a+b+c}{b-a}$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190505,
        "difficulty": null
      },
      {
        "id": "42415096447f61db72fd3a02d099ffc7f538d9044425eceb1da3013480918b7c",
        "domain": "numina",
        "question": "7.2. In Solar City, 6 dwarfs eat donuts daily, 8 dwarfs eat donuts every other day, and the rest do not eat donuts at all. Yesterday, 11 dwarfs ate donuts. How many dwarfs will eat donuts today?",
        "answer": "9",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 12316,
        "difficulty": null
      },
      {
        "id": "269898d9e211fb44533ad3529ad4198044c2e6083593dbc49e2bd89e8ba996a0",
        "domain": "numina",
        "question": "9. (1990 Hungarian Mathematical Olympiad) For any positive integer $q_{0}$, consider the sequence $q_{1}, q_{2}, \\cdots, q_{n}$ defined by $q_{i}=\\left(q_{i-1}-1\\right)^{3}+$ $3(i=1,2, \\cdots, n)$. If each $q_{i}(i=1,2, \\cdots, n)$ is a prime, find the largest possible value of $n$.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112153,
        "difficulty": null
      },
      {
        "id": "3f0975f77c7f7211f09be4fd601af25b359690153c2388920baef7793005960f",
        "domain": "numina",
        "question": "Task 2 - 130912 Someone wants to create a new mixture with a water content of $98 \\%$ from a mixture that consists of $99 \\%$ water, by removing water from the original mixture.\n\nDetermine how much percent of the water content in the original mixture he must remove for this purpose in total.",
        "answer": "51",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184520,
        "difficulty": null
      },
      {
        "id": "ca14c01d63b4943e58fd1c7ccc057e8a516568ede584ddc956ce059ae9629d46",
        "domain": "numina",
        "question": "153. a) Find all three-digit numbers that are equal to the sum of the factorials of their digits.\n\nb) Find all integers that are equal to the sum of the squares of their digits.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 29422,
        "difficulty": null
      },
      {
        "id": "d6d8996eed8a9b7326e151314d16cb09ad03ffbf4fffae047d0997d53a46528a",
        "domain": "numina",
        "question": "Some unit-radius circles are drawn on the plane, and we color the center of each circle blue. We mark some points on the circumference of the circles in red such that exactly 2 red points lie on each circumference. What is the maximum number of blue points if there are a total of 25 colored points?",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 169823,
        "difficulty": null
      },
      {
        "id": "7b672d98764c6024519ca77ec7150a9fec3526959499cbe2440affc544793598",
        "domain": "numina",
        "question": "3. Write down all three-digit numbers whose sum of digits is equal to 9. How many are there?",
        "answer": "45",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87092,
        "difficulty": null
      },
      {
        "id": "45b1f6770305a44ccb20f7d0ae85775cae99cdced5fa2ee794a8ee551e1ed4b5",
        "domain": "numina",
        "question": "[ Sum of angles in a triangle. Theorem about the exterior angle. ] [ Criteria and properties of an isosceles triangle. ]\n\nIn triangle $A B C$, angle $C$ is three times angle $A$. A point $D$ is taken on side $A B$ such that $B D=B C$.\n\nFind $C D$, if $A D=4$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66595,
        "difficulty": null
      },
      {
        "id": "cee6968f6db4ab1005d9f6a7dd150481599832a6432974259f61d8052976a67b",
        "domain": "numina",
        "question": "[ Arithmetic. Mental calculation, etc.]\n\nThe sum of the subtrahend, minuend, and difference is 2016. Find the minuend.\n\n#",
        "answer": "1008",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 70663,
        "difficulty": null
      },
      {
        "id": "46520dd2023ba6dfd1079c8578c827a23888fdcd345ca36305fd2ac3a26c5db4",
        "domain": "numina",
        "question": "6. Find all positive integers $n$ such that $3^{2 n}+3 n^{2}+7$ is a perfect square.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 180317,
        "difficulty": null
      },
      {
        "id": "0693af1618c94a38773c7ff77caa1a369eeebc3c67d0233490ed3aa1096e64c6",
        "domain": "numina",
        "question": "6.53. $\\lim _{x \\rightarrow \\infty} \\frac{x}{e^{x}}$.",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 33405,
        "difficulty": null
      },
      {
        "id": "83621b4668d9d0d6f4fb369eeb71f9bba2ea2ca653d28b55daf6ec175a6fb32e",
        "domain": "numina",
        "question": "74. In a rectangular coordinate system, there are four points $A(-1,1)$, $B(5,0)$, $C(3,-3)$, and $D(-3,-2)$. After moving point $A$ down by one unit and moving point $C$ up by one unit, the area of the new quadrilateral $ABCD$ is . $\\qquad$",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 104320,
        "difficulty": null
      },
      {
        "id": "47428014167650e53f986440554f8071ef197b7d2cb42cbdb0472d20d3a6f9f0",
        "domain": "numina",
        "question": "4. Let $p(x)=2 x^{3}-3 x^{2}+1$. How many squares of integers are among the numbers $p(1), p(2), \\ldots, p(2016)$?",
        "answer": "32",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12997,
        "difficulty": null
      },
      {
        "id": "1f5ecc18b2a1907276f10f4e85d5b64de11c5bda01e23b826a1e8db33472798e",
        "domain": "numina",
        "question": "5. Let for all real numbers $x$, the function $f(x)$ satisfies $x f(x)=2 f(1-x)+1$.\nThen $f(5)=$ $\\qquad$",
        "answer": "\\frac{1}{12}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122199,
        "difficulty": null
      },
      {
        "id": "f3e55246089fae97d849490698288ddb548a1002e7be7390919368387ba9d5b7",
        "domain": "numina",
        "question": "15. Five Bunchkins sit in a horizontal field. No three of the Bunchkins are sitting in a straight line. Each Bunchkin knows the four distances between her and each of the others. Each Bunchkin calculates and then announces the total of these distances. These totals are $17,43,56,66$ and 76. A straight line is painted joining each pair of Bunchkins. What is the total length of paint required?",
        "answer": "129",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 156485,
        "difficulty": null
      },
      {
        "id": "41210bc1f617d8a81b028bedadfb62767a65179070988fe9f82ac4fce32cf90e",
        "domain": "numina",
        "question": "3. Let the function $f(x)=\\frac{(x+1)^{2}+\\sin x}{x^{2}+1}$ have the maximum value and minimum value as $M$ and $N$, respectively. Then $M+N=$ $\\qquad$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129801,
        "difficulty": null
      },
      {
        "id": "3df78f1090c798c95d6fac177a6e4e8428d0629af0896693b61a60b1ceb98647",
        "domain": "numina",
        "question": "I1.2 Let $20112011=a P^{5}+b P^{4}+c P^{3}+d P^{2}+e P+f$, where $a, b, c, d, e$ and $f$ are integers and $0 \\leq a$, $b, c, d, e, f<P$. If $Q=a+b+c+d+e+f$, find the value of $Q$.",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153280,
        "difficulty": null
      },
      {
        "id": "498d1fc4ec23b5ca520230cb41d86586645f3e0aca2cedc850c26c8f15779355",
        "domain": "numina",
        "question": "19. (Mathematics for Middle School, Issue 4, 2004, Olympiad Problem 137) Try to find the maximum number of elements in a set $S$ that satisfies the following conditions:\n(1) Each element in $S$ is a positive integer not exceeding 100;\n(2) For any two different elements $a$ and $b$ in $S$, there exists another element $c$ in $S$ such that the greatest common divisor of $a+b$ and $c$ is 1;\n(3) For any two different elements $a$ and $b$ in $S$, there exists another element $c$ in $S$ such that the greatest common divisor of $a+b$ and $c$ is greater than 1.",
        "answer": "50",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 110131,
        "difficulty": null
      },
      {
        "id": "37bc135646f4e3f89a744de39bb3fcd6eee7e04bc8ca74f79d414279654bd640",
        "domain": "numina",
        "question": "6. Let $S$ be the set of all rational numbers in the interval $\\left(0, \\frac{5}{8}\\right)$, for the fraction $\\frac{q}{p} \\in S,(p, q)=1$, define the function $f\\left(\\frac{q}{p}\\right)=\\frac{q+1}{p}$. Then the number of roots of $f(x)=\\frac{2}{3}$ in the set $S$ is $\\qquad$",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 143424,
        "difficulty": null
      },
      {
        "id": "bb343b871004bbdf5aa94ccdc54c9214242083d03ea70236e085218d17277ca9",
        "domain": "numina",
        "question": "10. For what value of $a$ does the inequality $\\log _{\\frac{1}{4}}\\left(\\sqrt{x^{2}+a x+5}+1\\right) \\cdot \\log _{5}\\left(x^{2}+a x+6\\right)+\\log _{a} 3 \\geqslant 0$ have exactly one solution?",
        "answer": "2",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 136614,
        "difficulty": null
      },
      {
        "id": "f53236dc9b3b42d34dd9cba795a5d3b67891ff17a2c5eed11ee489d988803986",
        "domain": "numina",
        "question": "4B. If $\\cos x+\\cos ^{2} x+\\cos ^{3} x=1$, calculate the value of the expression\n\n$$\n\\sin ^{6} x-4 \\sin ^{4} x+8 \\sin ^{2} x\n$$",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190170,
        "difficulty": null
      },
      {
        "id": "71d31ac6e0f283bf621f810540fddb5260a6f5a9e11d0d7811a592994bc9ef63",
        "domain": "numina",
        "question": "Example 5 Let the set $A=\\{1,2,3,4,5,6\\}$, and the mapping $f: A \\rightarrow A$, such that its third composition $f \\cdot f \\cdot f$ is the identity mapping. How many such $f$ are there?\n(1996 Japan Mathematical Olympiad Preliminary)",
        "answer": "81",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 114456,
        "difficulty": null
      },
      {
        "id": "8fd19d6aed1ce0ad10702a1fa3706ce6c53468fe0a821edeb9740c8587870982",
        "domain": "numina",
        "question": "8. (10 points) Place seven consecutive natural numbers at the intersection points $A, B, C, D, E, F, G$ of five circles, such that the sum of the numbers on each circle is equal. If all the numbers filled in are greater than 0 and no more than 10, then the number filled at point $G$ is.",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 145640,
        "difficulty": null
      },
      {
        "id": "03da390b8a889ea69f19361a4f103ea4b75f9998c98eb42258605c35a920a6c3",
        "domain": "numina",
        "question": "Task 1. (5 points) Find $\\frac{a^{12}+4096}{64 a^{6}}$, if $\\frac{a}{2}-\\frac{2}{a}=3$.\n\n#",
        "answer": "1298",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 7187,
        "difficulty": null
      },
      {
        "id": "28236b68f4bbf7e4ef1573736dfb73d60c10aa81cab2b608f8b1b766097803d6",
        "domain": "numina",
        "question": "13. (12 points) $A$ and $B$ are two places where a bus is dispatched every 5 minutes, traveling at a uniform speed towards each other, and all buses travel at the same speed; two people, Jia and Yi, start from $A$ and $B$ respectively, traveling towards each other at a uniform speed; 5 minutes after Jia and Yi start, the first bus departs from both places; when Jia and Yi meet, Jia is caught up by the 9th bus from $A$, and Yi is also caught up by the 6th bus from $B$; when Yi arrives at $A$, he is caught up by the 8th bus from $B$, and at this time, Jia is still 21 kilometers away from $B$. What is Yi's speed in kilometers per hour?",
        "answer": "27",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100016,
        "difficulty": null
      },
      {
        "id": "c5ae09cb367af26cf2c6b6c4a13ea76e5f9e749c3b09334923cf28424123a9f4",
        "domain": "numina",
        "question": "The distinct prime factors of 18 are 2 and 3 . What is the sum of all of the distinct prime factors of $4446 ?$",
        "answer": "37",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194458,
        "difficulty": null
      },
      {
        "id": "39290b320c25e755c9778cacdaf5ed065ac8e9fa30bc5fd9d7fd617c1177805f",
        "domain": "numina",
        "question": "2. From $1,2,3, \\cdots, 14$ select $a_{1}, a_{2}, a_{3}$ in ascending order, such that $a_{2}-a_{1} \\geqslant 3$, $a_{3}-a_{2} \\geqslant 3$ are simultaneously satisfied. The number of all different selection methods that meet the above requirements is $\\qquad$.",
        "answer": "120",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144063,
        "difficulty": null
      },
      {
        "id": "7d4bcc0834cb81d370f699347061522a0d1f2088ec58e3da1179478d23517592",
        "domain": "numina",
        "question": "We start with 5000 forints in our pocket to buy gifts. We enter three stores. In each store, we like a gift item, which we will buy if we can afford it. The prices of the items in each store, independently of each other, are 1000, 1500, or 2000 Ft with a probability of $\\frac{1}{3}$. What is the probability that we will manage to buy three gift items and still have money left?",
        "answer": "\\frac{17}{27}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 171273,
        "difficulty": null
      },
      {
        "id": "fc01fbadf9e17a79dffbb2b40796f662dc85f45cfb0b68fbcfde2ed7e9c432ce",
        "domain": "numina",
        "question": "1. All gnomes are divided into liars and knights. Liars always lie, and knights always tell the truth. On each cell of a $4 \\times 4$ board, there is a gnome. It is known that among them, there are both liars and knights. Each gnome stated: “Among my neighbors (by side) there are an equal number of liars and knights.” How many liars are there in total?",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14724,
        "difficulty": null
      },
      {
        "id": "c7f8982b3e15d2943a00aab7731f4fa38bb57261200eb055c71152b418cc06e7",
        "domain": "numina",
        "question": "6. A rectangle $3 \\times 100$ consists of 300 squares $1 \\times 1$. What is the maximum number of diagonals that can be drawn in the squares so that no two diagonals share a common endpoint? (In one square, two diagonals can be drawn without sharing endpoints. Common internal points are allowed.)",
        "answer": "200",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18288,
        "difficulty": null
      },
      {
        "id": "5914c65906de5c5eb5416379916fb78a35a5e0580909eac493ab1bd42c2a1c92",
        "domain": "numina",
        "question": "[ Percentage and ratio problems ]\n\nHow many pounds of grain need to be milled, so that after paying for the work at $-10 \\%$ of the milling - exactly 100 pounds of flour remain? There are no losses during milling.\n\n#",
        "answer": "111+\\frac{1}{9}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69086,
        "difficulty": null
      },
      {
        "id": "9584d5872869c5a031345a45758a9669e8bfe6a589c756cd68d13f51566a88c2",
        "domain": "numina",
        "question": "## Subject IV. (20 points)\n\nAmaya and Keiko are two girls from Osaka. They live on the same floor in a building with two staircases, each with 5 apartments per floor. At the ground floor of the building are shops.\n\nThe apartments are numbered in ascending order, starting from the first floor. Amaya lives in apartment 28, and Keiko in apartment 164. How many floors does the building have?\n\nProf. Sorin Borodi, \"Alexandru Papiu Ilarian\" Theoretical High School, Dej\n\nAll subjects are mandatory. 10 points are awarded by default.\n\nEffective working time - 2 hours.\n\n## Grading Scale for Grade 5 (OLM 2013 - Local Stage)\n\n## Of. $10 \\mathrm{p}$",
        "answer": "27",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 179905,
        "difficulty": null
      },
      {
        "id": "11545cd5e5dfac0a9f675774e4fd3857f5e67b0ec9efe6381564c65b6d8742ae",
        "domain": "numina",
        "question": "## Task $8 / 90$\n\nLet $n$ be a natural number with exactly $m$ digits in decimal notation $(m \\geq 2)$, which is divisible by 11. By reversing the sequence of digits, the natural number $n^{\\prime}$ (not necessarily with exactly $m$ digits) is formed. How many sums $n+n^{\\prime}$ are divisible by 11?",
        "answer": "9\\cdot\\frac{10^{-1}+(-1)^{}}{11}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182869,
        "difficulty": null
      },
      {
        "id": "9b67329a62c95d527f542b100d764ade267f1a7f659e9781d0e48b0610e643c6",
        "domain": "numina",
        "question": "6. 15 cards, each card has 3 different Chinese characters, any 2 cards do not have exactly the same characters; among any 6 cards, there must be 2 cards that have a common character. Question: What is the maximum number of different Chinese characters on these 15 cards?",
        "answer": "35",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 98883,
        "difficulty": null
      },
      {
        "id": "055b2477f32406ddfee895ae8471b4c60422476029dcd190b9fa9ba768899fd6",
        "domain": "numina",
        "question": "41. The road from home to school takes Seryozha 30 minutes. Once on the way, he remembered that he had forgotten a pen at home. Seryozha knew that if he continued on to school at the same speed, he would arrive there 9 minutes before the bell, but if he returned home for the pen, he would, walking at the same speed, be 11 minutes late for the start of the lesson. What part of the way had he walked?",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39531,
        "difficulty": null
      },
      {
        "id": "dbcef54cae80489a171df747ec1fa7e36634e55f28db99a5c165ee6d1bed0cca",
        "domain": "numina",
        "question": "In a right-angled triangle, the height drawn to the hypotenuse is $12 \\mathrm{~cm}$; the difference between the two segments of the hypotenuse is $7 \\mathrm{~cm}$. What are the lengths of the sides of the triangle?",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 175363,
        "difficulty": null
      },
      {
        "id": "ba1d459658287ee5557b960931a98915b2406ef48295e9ad70c43d4386ea20e4",
        "domain": "numina",
        "question": "1.2. In triangle $ABC$ with angle $BAC$ equal to $24^{\\circ}$, points $X$ and $Y$ are taken on sides $AB$ and $AC$ respectively. The circle centered at $Y$ and passing through $A$ also passes through $X$, and the circle centered at $X$ and passing through $B$ also passes through $C$ and $Y$. Find $\\angle ABC$.",
        "answer": "54",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 162023,
        "difficulty": null
      },
      {
        "id": "cad21b2d99e6c8d4ed70917edbf3b9e48968c42eed963d33eabb12c4b51d9d73",
        "domain": "numina",
        "question": "1. The quadratic trinomial $f(x)=a x^{2}+b x+c$ has exactly one root, the quadratic trinomial $f(3 x+2)-2 f(2 x-1)$ also has exactly one root. Find the root of the trinomial $f(x)$.",
        "answer": "-7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 8351,
        "difficulty": null
      },
      {
        "id": "6793edcd9e36d031694df6b27a8f7319bbfc30896a8cda2f052ad17dee2b93ac",
        "domain": "numina",
        "question": "10. [7] Let $\\omega=\\cos \\frac{2 \\pi}{727}+i \\sin \\frac{2 \\pi}{727}$. The imaginary part of the complex number\n$$\n\\prod_{k=8}^{13}\\left(1+\\omega^{3^{k-1}}+\\omega^{2 \\cdot 3^{k-1}}\\right)\n$$\nis equal to $\\sin \\alpha$ for some angle $\\alpha$ between $-\\frac{\\pi}{2}$ and $\\frac{\\pi}{2}$, inclusive. Find $\\alpha$.",
        "answer": "\\frac{12\\pi}{727}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151494,
        "difficulty": null
      },
      {
        "id": "63f19910c40b26aea3e63b69e9ba00c203214e56104961e00e8851ff237779c0",
        "domain": "numina",
        "question": "A pyramid has a base that is a right-angled quadrilateral, one side of which is $a=6 \\mathrm{~cm}$, the other side is $b=8 \\mathrm{~cm}$; each edge of the pyramid is $c=12 \\mathrm{~cm}$. At what distance from the apex must the pyramid be cut by a plane parallel to the base so that the volumes of the resulting parts are equal?",
        "answer": "8.658",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174435,
        "difficulty": null
      },
      {
        "id": "8c1c0f381eb1e43fdfb4ba38b0fd21c38de1049061573556c2d3ada364f309a3",
        "domain": "numina",
        "question": "## Task 2\n\nTwo birds are sitting on a branch $8 \\mathrm{~m}$ apart.\n\nNow one bird hops 1 meter towards the other. The other bird then hops 2 meters towards the first.\n\nHow far apart are the two birds now?",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186314,
        "difficulty": null
      },
      {
        "id": "db2eb3a6528b812e32d853ef3cf704795eed69ba679591b15f0bff362ef199ec",
        "domain": "numina",
        "question": "Simplify the following expression as much as possible:\n\n$$\n\\left(\\frac{\\sqrt{1+a}}{\\sqrt{1+a}-\\sqrt{1-a}}+\\frac{1-a}{\\sqrt{1-a^{2}}-1+a}\\right)\\left(\\sqrt{\\frac{1}{a^{2}}-1}-\\frac{1}{a}\\right)\n$$\n\nwhere $0<a<1$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 175241,
        "difficulty": null
      },
      {
        "id": "fcf734d7ae71e26ffb7e475fe3027eaf78d2974e131cbe9c7eb8e057b3c2a856",
        "domain": "numina",
        "question": "13. A. For any real number $x$, let $\\lfloor x\\rfloor$ denote the greatest integer not exceeding the real number $x$, $\\{x\\}=x-\\lfloor x\\rfloor$, and $\\lceil x\\rceil$ denote the smallest integer not less than the real number $x$. If $x_{1}, x_{2}, \\cdots, x_{m}$ $\\left(0 \\leqslant x_{1}<x_{2}<\\cdots<x_{m} \\leqslant 10\\right)$ are all the real numbers between 0 and 10 that satisfy the equation $\\lfloor x\\rfloor\\{x\\}\\lceil x\\rceil=1$, find the value of $x_{1}+$ $x_{2}+\\cdots+x_{m}$.",
        "answer": "\\frac{459}{10}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 109831,
        "difficulty": null
      },
      {
        "id": "45fd7eca2e2afbf0404b6750025299c186f2de908c18dd77596f8bc8d64432bb",
        "domain": "numina",
        "question": "(1) Given the function $f(x)=x^{3}+a x^{2}+x+1(a \\in \\mathbf{R})$ is decreasing in the interval $\\left(-\\frac{2}{3},-\\frac{1}{3}\\right)$ and increasing in the interval $\\left(-\\frac{1}{3},+\\infty\\right)$, then $a=$",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 93274,
        "difficulty": null
      },
      {
        "id": "f1b3908e3a722b3fbbc145364a3624710fdf2e9e1f348ca4be4fbef3bf90f6dd",
        "domain": "numina",
        "question": "6. The solution to the equation $2^{x^{2}-2 x}+3^{x^{2}-2 x}+x^{2}-2 x+\\frac{1}{6}=0$ is",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117582,
        "difficulty": null
      },
      {
        "id": "c73734d5fc7fae55873ded334e0dd7f18ab1b9d3251440abeca93964177c5d35",
        "domain": "numina",
        "question": "Problem 8.6. For quadrilateral $ABCD$, it is known that $AB=BD, \\angle ABD=\\angle DBC, \\angle BCD=$ $90^{\\circ}$. A point $E$ is marked on segment $BC$ such that $AD=DE$. What is the length of segment $BD$, if it is known that $BE=7, EC=5$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c49ecf27446af41a8bcdg-30.jpg?height=476&width=517&top_left_y=91&top_left_x=468)",
        "answer": "17",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17851,
        "difficulty": null
      },
      {
        "id": "d2c76378f21e2667e9b38383c6b4cbd04c6c331fe024db4438777181218d4adb",
        "domain": "numina",
        "question": "5. [5 points] On a plane with a given rectangular Cartesian coordinate system, a square is drawn with vertices at points $(0 ; 0),(0 ; 69),(69 ; 69)$, and ( $69 ; 0)$. Find the number of ways to choose two grid nodes inside this square (not including its boundary) such that at least one of these nodes lies on one of the lines $y=x$ or $y=69-x$, but neither of the chosen nodes lies on any line parallel to any of the coordinate axes.",
        "answer": "601460",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 3799,
        "difficulty": null
      },
      {
        "id": "50c71187245976367ca16c84c309382a28d27136f09b3e6ebdab4c5e6c921fae",
        "domain": "numina",
        "question": "Determine all positive real numbers $x$ and $y$ satisfying the equation\n\n$$\nx+y+\\frac{1}{x}+\\frac{1}{y}+4=2 \\cdot(\\sqrt{2 x+1}+\\sqrt{2 y+1}) .\n$$",
        "answer": "1+\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 72916,
        "difficulty": null
      },
      {
        "id": "08b899f6394b20e2944ae463ce44799f2ced296959d0159b7eab40a7ba985235",
        "domain": "numina",
        "question": "Andjans A.\n\nIn a table with $m$ rows and $n$ columns, a horizontal move is defined as a permutation of the elements in the table such that each element remains in the same row it was in before the permutation; similarly, a vertical move is defined (\"row\" in the previous definition is replaced with \"column\"). Determine such a $k$ that any permutation of the elements in the table can be achieved in $k$ moves (of any kind), but there exists a permutation that cannot be achieved in fewer moves.",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 56915,
        "difficulty": null
      },
      {
        "id": "70193db7e2c53b497d56d5e50a261e0150ab96e93869582ae8ba393029cdc532",
        "domain": "numina",
        "question": "3. If $z$ is a complex number with a non-zero real part, then the minimum value of $\\frac{\\operatorname{Re}\\left(z^{4}\\right)}{(\\operatorname{Re}(z))^{4}}$ is $\\qquad$",
        "answer": "-8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 137575,
        "difficulty": null
      },
      {
        "id": "679c9de4f2f8a107e632db02c9b330253141e27bb4112338db3d762c0db1ec1b",
        "domain": "numina",
        "question": "18. There are three types of people, A, B, and C, totaling 25 people. A type people always tell the truth, B type people always lie, and C type people alternate between telling the truth and lying (for example, if a C type person tells the truth this time, the next thing they say will definitely be a lie, and the one after that will be the truth again).\nThe priest asked everyone, “Are you an A type person?” 17 people answered “yes”.\nThe priest then asked everyone, “Are you a C type person?” 12 people answered “yes”.\nThe priest then asked everyone, “Are you a B type person?” 8 people answered “yes”.\n\nAmong these 25 people, ( ) are C type people.",
        "answer": "16",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 139582,
        "difficulty": null
      },
      {
        "id": "0e48728bde10600198f695e0e9fc09cd48fb1ced17ec42949cbaba0ce95b3a39",
        "domain": "numina",
        "question": "3. [4] Let $A B C$ be a triangle with $\\angle B A C=90^{\\circ}$. A circle is tangent to the sides $A B$ and $A C$ at $X$ and $Y$ respectively, such that the points on the circle diametrically opposite $X$ and $Y$ both lie on the side $B C$. Given that $A B=6$, find the area of the portion of the circle that lies outside the triangle.",
        "answer": "\\pi-2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150949,
        "difficulty": null
      },
      {
        "id": "f66b56166932fa9a93e63862c47e4efd213cead681f13c25e5c0a77c9ef47327",
        "domain": "numina",
        "question": "I3.1 Given that $\\frac{4}{\\sqrt{6}+\\sqrt{2}}-\\frac{1}{\\sqrt{3}+\\sqrt{2}}=\\sqrt{a}-\\sqrt{b}$. If $m=a-b$, find the value of $m$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153153,
        "difficulty": null
      },
      {
        "id": "4b0a9793b1501af1b8233e255ca0144a68d13757607ab2825777e9eabd2ff04a",
        "domain": "numina",
        "question": "Task 1. A grass path surrounds a rectangular asphalt playground with dimensions $36 \\mathrm{dm}$ and $12 \\mathrm{~m}$ (as shown in the diagram). The width $d$ of the path is equal to a quarter of the semiperimeter of the playground. How many meters of wire are needed to fence the grass path around the playground, if it is surrounded by 5 rows of wire? How many posts are needed to secure the fence, if the distance between any two adjacent posts is $6 \\mathrm{dm}$?",
        "answer": "468",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188968,
        "difficulty": null
      },
      {
        "id": "d2a29aad720e3aa5b2cb7bec89f37eba6062d581422061bb8ff746c748b05eda",
        "domain": "numina",
        "question": "I2.4 The sum of $2 C$ consecutive even numbers is 1170 . If $D$ is the largest of them, find $D$.",
        "answer": "68",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152733,
        "difficulty": null
      },
      {
        "id": "3e6a082fcd3cc016c36a2f703c4e482c4ff4a1d4809eb1d8749227198d37fe60",
        "domain": "numina",
        "question": "Booting D..\n\nKnower came to visit the twin brothers Vintik and Shpuntik, knowing that one of them never tells the truth, and asked one of them: \"Are you Vintik?\" \"Yes,\" — answered the one. When Knower asked the same question of the second, he received just as clear an answer and immediately determined who was who.\n\nWho was called Vintik?",
        "answer": "1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56631,
        "difficulty": null
      },
      {
        "id": "dd25984ace00dd76a18cfff1b9a658c5745fc460e1623ef8b6209b798cdf8698",
        "domain": "numina",
        "question": "12. If: (1) $a, b, c, d$ all belong to $\\{1,2,3,4\\}$;\n(2) $a \\neq b, b \\neq c, c \\neq d, d \\neq a$;\n(3) $a$ is the smallest value among $a, b, c, d$,\n\nthen, the number of different four-digit numbers $\\overline{a b c d}$ that can be formed is",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 121355,
        "difficulty": null
      },
      {
        "id": "8112d43d44acc80f23efde396de4c8e2cef1755a53271046f324066be6d28deb",
        "domain": "numina",
        "question": "6. (10 points) The distance between $A$ and $B$ is 100 kilometers. Car A takes $m$ hours to travel from $A$ to $B$, and Car B takes $n$ hours to travel from $A$ to $B$, where $m, n$ are integers. Now, Car A starts from $A$ and Car B starts from $B$ at the same time, heading towards each other, and they meet at point $C$ after 5 hours. If Car A has already traveled half of its distance, then the distance from $C$ to $A$ is $\\qquad$ kilometers.",
        "answer": "83\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103600,
        "difficulty": null
      },
      {
        "id": "c2ee3650bbf3ac08b1e8a2e241f440eca56824fd732316c191984851db2d7fa1",
        "domain": "numina",
        "question": "78 Given $x \\geqslant 0, y \\geqslant 0$, and $x+y \\leqslant 2 \\pi$, try to find the maximum and minimum values of the function $f(x, y)=\\sin x+\\sin y- \\sin (x+y)$.",
        "answer": "\\frac{3\\sqrt{3}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107786,
        "difficulty": null
      },
      {
        "id": "13a02752ad376869de4b12e0de22cc7d4cab094bf77bb46e2c2a63b9133afe8a",
        "domain": "numina",
        "question": "Example 3 Given that $f(x)$ is a function defined on $\\mathbf{R}$, $f(1)=1$. And for any $x \\in \\mathbf{R}$, there are $f(x+5) \\geqslant f(x)+5, f(x+1) \\leqslant f(x)+1$. If $g(x)=f(x)+1-x$, find $g(2002)$.\n(2002 National High School League Question)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 125762,
        "difficulty": null
      },
      {
        "id": "26c2adf2b8a9c4ebb71ed388186aef34286f4e9ed401f30555c364d4a75604c0",
        "domain": "numina",
        "question": "## Task 12\n\nIn the three schools of a city, there are a total of 63 Pioneer groups. From each group, 4 Pioneers can go to the Pioneer meeting. They are picked up by 6 equally large buses.\n\nHow many Pioneers are in each bus?",
        "answer": "42",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 186738,
        "difficulty": null
      },
      {
        "id": "5a26fa26bb24612b5a0792b908be5933603b17d4f8292cd4c70ca8309f0bce03",
        "domain": "numina",
        "question": "7. Consider a positive integer,\n$$\n\\mathrm{N}=9+99+999+\\ldots \\ldots+\\underset{2018}{999 \\ldots 9}\n$$\n\nHow many times does the digit 1 occur in its decimal representation?",
        "answer": "2014",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153923,
        "difficulty": null
      },
      {
        "id": "18077bbe90c7e2901d0dcff404ebae3998bc3d302f679c9d1f9b351e60b447be",
        "domain": "numina",
        "question": "2. Determine the maximum value of the expression\n\n$$\n\\log _{2}^{4} x+12 \\log _{2}^{2} x \\cdot \\log _{2}\\left(\\frac{8}{x}\\right)\n$$\n\nif the variable $x$ varies on the interval $[1,64]$.",
        "answer": "81",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189539,
        "difficulty": null
      },
      {
        "id": "b8fce369247753d9fc1e3bc6c9c0e6b3728623c4cf9dbd799954c7059f4b11b1",
        "domain": "numina",
        "question": "Example 6. Find $\\lim _{x \\rightarrow 3} \\frac{\\ln \\left(x^{2}-5 x+7\\right)}{x-3}$.",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32614,
        "difficulty": null
      },
      {
        "id": "604f072c20aabf47ebde2118a381d8cc99a634a60707b83e88240802b3e13d59",
        "domain": "numina",
        "question": "6. A $41 \\times 41$ battlefield is divided into 1681 unit squares. A tank is hidden in one of these unit squares. A bomber aircraft bombs the battlefield. Each bombing targets one unit square. If the tank's square is not bombed, the tank remains in place; if it is bombed, the tank must move to an adjacent unit square (two squares are considered \"adjacent\" if they share a common edge). Destroying the tank requires hitting it twice. Throughout the process, the bomber pilot has no knowledge of the tank's location or whether it has been hit. Determine the minimum number of bombings required to ensure the tank is destroyed.",
        "answer": "2521",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 96560,
        "difficulty": null
      },
      {
        "id": "c042764baa2733f0060bf7f892e2b8a289e13c3b00203987d21e2cdf8d3c74e8",
        "domain": "numina",
        "question": "3. Solve the equation $\\frac{\\sqrt{(-x)^{2}}+(\\sqrt{-x})^{2}}{x^{2}+(-x)^{2}}=\\frac{1}{2016}$\n\nANSWER: $x=-2016$.",
        "answer": "-2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5614,
        "difficulty": null
      },
      {
        "id": "784446a6605218ab728f12e827f92c59b36284ff3e1b5898d9908da5216983a8",
        "domain": "numina",
        "question": "12. There are 8 black, 8 white, and 8 yellow chopsticks mixed together. In the dark, you want to take out two pairs of chopsticks of different colors. How many chopsticks do you need to take out to ensure you meet the requirement?",
        "answer": "11",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 130730,
        "difficulty": null
      },
      {
        "id": "e23c5fbdd69ccca3c799ef13de13550ed38159f35071b0b89d03363bddcb4159",
        "domain": "numina",
        "question": "3.1. Alina and Masha wanted to create an interesting version of the school tour olympiad. Masha proposed several problems and rejected every second problem of Alina's (exactly half), Alina also proposed several problems and did not reject only every third problem of Masha's (exactly a third). In the end, there were 10 problems, and initially, 27 were proposed. How many more problems did Masha propose than Alina?",
        "answer": "15",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17601,
        "difficulty": null
      },
      {
        "id": "6aedd06587f2cd26f36e00c7523059fa6beb9394eb65ef2d31c16d111ccaa180",
        "domain": "numina",
        "question": "B1 Give the smallest positive integer that is divisible by 26, ends in 26, and for which the sum of the digits equals 26.",
        "answer": "46826",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85798,
        "difficulty": null
      },
      {
        "id": "171ba07d5930c222297ffd4543f166ba5063ac94704a70d4a204c98524701fcf",
        "domain": "numina",
        "question": "2、Given $A ※ B=(A+3) \\times(B-2)$. Then $12 ※ 17=$ $\\qquad$ -",
        "answer": "225",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 113882,
        "difficulty": null
      },
      {
        "id": "7130d4fa9d826e6029ed30b61c826ecad1211b61e7262b248b557221c91e83de",
        "domain": "numina",
        "question": "Problem 8.7. Given an isosceles triangle $A B C$, where $A B=A C$ and $\\angle A B C=53^{\\circ}$. Point $K$ is such that $C$ is the midpoint of segment $A K$. Point $M$ is chosen such that:\n\n- $B$ and $M$ are on the same side of line $A C$;\n- $K M=A B$\n- angle $M A K$ is the maximum possible.\n\nHow many degrees does angle $B A M$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4984b677816e04f7f1c9g-27.jpg?height=400&width=301&top_left_y=224&top_left_x=576)",
        "answer": "44",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14848,
        "difficulty": null
      },
      {
        "id": "959b079d138d42f3ce591b67019e84e4f735a171c4c29a2392837a0034bc6515",
        "domain": "numina",
        "question": "3. (7 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $15$, and $\\sin \\alpha = \\frac{\\sqrt{21}}{5}$?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19498,
        "difficulty": null
      },
      {
        "id": "58047525fc21fd9a414a1dd11f9c1edc6faf44b88dcf6dfdcd6ee5bf8e27250f",
        "domain": "numina",
        "question": "626. Find $f^{\\prime}(0)$, if $f(x)=e^{x} \\arcsin x+\\operatorname{arctg} x$.",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30524,
        "difficulty": null
      },
      {
        "id": "b699f53d6f318dbf126bf88d0a078d6a6b48a8b76a1b1eeb97e14797c40ffb9d",
        "domain": "numina",
        "question": "7. Given positive rational numbers $a, b$, (both not 0, and $a+b+c=abc$ then $\\frac{a}{b}+\\frac{a}{c}+\\frac{b}{a}+$ $\\frac{b}{c}+\\frac{c}{a}+\\frac{c}{b}-ab-bc-ca=$ $\\qquad$ .",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97286,
        "difficulty": null
      },
      {
        "id": "9e6483bdf59f993ef7222c8fb462fa811bff953115d447a8c43976971b1bc77d",
        "domain": "numina",
        "question": "## Task B-4.6.\n\nAn elevator can transport a maximum of eight people in one ride. Eleven residents, including two who are feuding, are waiting for the elevator, and they are all going to the same floor. What is the probability that the feuding residents will be in the elevator if all residents must be transported by the elevator in the minimum number of rides?",
        "answer": "\\frac{155}{319}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88478,
        "difficulty": null
      },
      {
        "id": "a22584a7d1ad3352fb491007946d954c814c08cafb2d6fcd5f6b12da9d7f589f",
        "domain": "numina",
        "question": "Now we have three types of salt water, A, B, and C, with concentrations of $5\\%$, $8\\%$, and $9\\%$ respectively, and weights of 60 grams, 60 grams, and 47 grams. We need to prepare 100 grams of salt water with a concentration of $7\\%$. Try to find out the maximum and minimum amount of salt water A that can be used. Please write down the sum of these two numbers as the answer to this question.",
        "answer": "84",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96094,
        "difficulty": null
      },
      {
        "id": "b472ed62c7a240dd9e015b550b7dcf232ccb0ad02a3e8af428356eb11d350b07",
        "domain": "numina",
        "question": "Example 8 A deck of tri-color cards consists of 32 cards in total, with 10 cards of each color: red, yellow, and blue, numbered $1, 2, \\cdots, 10$; there are also a big joker and a small joker, each numbered 0. Several cards are drawn from this deck, and the score is calculated according to the following rule: each card numbered $k$ is worth $2^{k}$ points. If the total score of these cards is 2004, they are called a \"good\" card group. Find the number of \"good\" card groups.",
        "answer": "1006009",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123738,
        "difficulty": null
      },
      {
        "id": "18a2d8f37f85a0f2542a590567ee581df296e6bc4a425efd5b209db62cb639e9",
        "domain": "numina",
        "question": "11.076. The height of the cone is equal to the diameter of its base. Find the ratio of the area of its base to the lateral surface area.",
        "answer": "\\frac{\\sqrt{5}}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51365,
        "difficulty": null
      },
      {
        "id": "a4707e92133141743f7f1502c15b975bdf590e9153e1eafa24255d84cd0b34fe",
        "domain": "numina",
        "question": "5. (6 points) There is a wall clock that strikes once at every hour, and it strikes as many times as the hour. At four o'clock, the clock takes 12 seconds to finish striking; then at twelve o'clock, it will take $\\qquad$ seconds to finish striking.",
        "answer": "44",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 130753,
        "difficulty": null
      },
      {
        "id": "528b7f9f01e07c50f637c527149f52859bd988d78186c5388f81cf4b42123d58",
        "domain": "numina",
        "question": "7.51 There are 1988 identical unit cubes, and they (all or part) are used to form three \"squares\" $A, B, C$ (i.e., three one-layer-high rectangular prisms with dimensions $a \\times a \\times 1, b \\times b \\times 1, c \\times c \\times 1$ where $a \\leqslant b \\leqslant c$). Now, place square $C$ on a flat surface, then place $B$ on top of $C$ so that each small block of $B$ is exactly on top of a small block of $C$, but the boundary of $B$ does not align with the sides of $C$. Finally, place $A$ on top of $B$ following the same principle, thus forming a \"three-story building\". When $a, b, c$ take what values, can the number of different \"three-story buildings\" that can be formed be maximized?",
        "answer": "345^2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 97282,
        "difficulty": null
      },
      {
        "id": "f42a7b4682e5cdb1f7b0d65cd6e1be99d4108ce72c62cd03046334b4c7bb7c3d",
        "domain": "numina",
        "question": "1-191 Write down all natural numbers from 1 to 1988 on a blackboard. Perform operations $A$ and $B$ on these numbers repeatedly: first $A$, then $B$, followed by $A$ again, and then $B$, and so on. Operation $A$ involves subtracting the same natural number from each number written on the blackboard (the number subtracted can be different for different instances of operation $A$). Operation $B$ involves erasing two numbers from the blackboard and writing down their sum. Operations $A$ and $B$ are performed in this sequence until, after a certain operation $B$, only one number remains on the blackboard, and it is non-negative. What is this number?",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 128996,
        "difficulty": null
      },
      {
        "id": "f1a594d0e55b689c06e18b4f8ea2e8f5ae89331408a34a7b047c00473e61898d",
        "domain": "numina",
        "question": "# Task № 7.3\n\n## Condition:\n\nArtist Ivan Konstantinovich decided to sell several of his paintings at the Broken Auction. The rules of the Broken Auction are as follows: first, Ivan Konstantinovich names a certain starting price for his painting, after which the participants who want to buy this painting begin to bid on it - increasing the price, but only by two times, - that is, the new price can only be twice the last one offered; the participant after whom no one dares to raise the price by another two times gets the painting; if no one bids on the painting, that is, no one offers a price higher than the initial one, the painting remains unsold. For each of his paintings, Ivan Konstantinovich decided to set the starting price at 1000 rubles. How many paintings did he sell at the Broken Auction if all his paintings were bought for different prices, and his total revenue from all the sold paintings amounted to 350000 rubles?",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16635,
        "difficulty": null
      },
      {
        "id": "facca4232243293a75b1680a1a32df8002c001ea9355fd6bd6b43709e93f7486",
        "domain": "numina",
        "question": "9. Given the line $l: x=m y+q, m \\neq 0$ intersects the ellipse $\\Gamma: 3 x^{2}+4 y^{2}=12$ at two distinct points $\\mathrm{A}$ and $\\mathrm{B}$. Let $\\mathrm{A}$'s reflection about the major axis of the ellipse be $A_{1}$, and $F$ be the right focus of the ellipse. Find the necessary and sufficient condition for $A_{1} 、 F 、 B$ to be collinear.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119584,
        "difficulty": null
      },
      {
        "id": "e58c7cf0b852c472aec16e7a455457ac434d09299b4e70ab48e77bd5feaa28c9",
        "domain": "numina",
        "question": "A spinner was created by drawing five radii from the centre of a circle. The first four radii divide the circle into four equal wedges. The fifth radius divides one of the wedges into two parts, one having twice the area of the other. The five wedges are labelled as pictured with the wedge labeled by 2 having twice the area of the wedge labeled by 1 . Determine the probability of spinning an odd number.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_d49dd011068fc0fd28d1g-1.jpg?height=333&width=320&top_left_y=1685&top_left_x=1466)",
        "answer": "\\frac{7}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194080,
        "difficulty": null
      },
      {
        "id": "1bd5cdeffb7504dff5decd0b7873cc06d012e5f1992c25bfaca7834fdd4e9aa4",
        "domain": "numina",
        "question": "88. Given that $x, y, z$ are three distinct non-zero natural numbers, if\n$$\n\\overline{x y y y y}+\\overline{x y y y}+\\overline{x y y}+\\overline{x y}+y=\\overline{y y y y z} .\n$$\n\nwhere $\\overline{x y y y y}$ and $\\overline{y y y y z}$ are both five-digit numbers, when $x+y$ is maximized, the corresponding value of $x+y+z$ is",
        "answer": "22",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 123058,
        "difficulty": null
      },
      {
        "id": "3cbd60899e18ac482ee66e07e307735e040d9c994126cccb9407f332cd382425",
        "domain": "numina",
        "question": "## Task 2.\n\nAnte wrote down a sequence $a_{1}, a_{2}, \\ldots, a_{2020}$ in which each of the numbers $1,2, \\ldots, 2020$ appears exactly once. Barbara wants to find out this sequence, and Ante gives her information through a series of exchanges.\n\nIn one exchange, Barbara writes down the numbers from 1 to 2020 on a piece of paper and connects some of them with lines, ensuring that each number is connected to at most one other number, and then hands the paper to Ante. Ante then writes down the numbers from 1 to 2020 on another piece of paper and for each pair of numbers $i$ and $j$ that Barbara connected with a line, he connects the numbers $a_{i}$ and $a_{j}$ with a line, and finally hands the paper to Barbara.\n\nHow many exchanges are needed at minimum for Barbara to determine Ante's sequence with certainty?",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89727,
        "difficulty": null
      },
      {
        "id": "41339b32684d5a08e8744299dff231b8812f30b5d242a001b27fa719aa326d35",
        "domain": "numina",
        "question": "8.2. Find the largest four-digit number in which all digits are different and which is divisible by any of its digits (don't forget to explain why it is the largest).",
        "answer": "9864",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 12047,
        "difficulty": null
      },
      {
        "id": "9dbacda099e83d0307792ac260aafb6c8ff4791790f5a098b3b43944acaea24c",
        "domain": "numina",
        "question": "7. When asked how old he was, a mathematician replied: \"If you subtract 5 from my age, divide the result by 5, and then subtract 5 again, you will get the number 5.\" How old is he?",
        "answer": "55",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86729,
        "difficulty": null
      },
      {
        "id": "622271a0c4ca60a1579579250df6e225ba8b6752ce807c6c1d568d11d4ff5d59",
        "domain": "numina",
        "question": "3. Given an isosceles triangle \\(ABC\\) with base \\(BC\\). On the extension of side \\(AC\\) beyond point \\(C\\), point \\(K\\) is marked, and a circle is inscribed in triangle \\(ABK\\) with center at point \\(I\\). A circle passing through points \\(B\\) and \\(I\\) is tangent to line \\(AB\\) at point \\(B\\). This circle intersects segment \\(BK\\) again at point \\(L\\). Find the angle between lines \\(IK\\) and \\(CL\\). Answer: \\(90^{\\circ}\\).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b7830eaf581def5094e2g-02.jpg?height=496&width=820&top_left_y=483&top_left_x=638)",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8436,
        "difficulty": null
      },
      {
        "id": "16d7575e64f51404e961c8c0ebc571494cb2130bd8b8f65d14fa50540fc0fa78",
        "domain": "numina",
        "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $6$ and $\\cos \\alpha = \\frac{2}{3}$?",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19403,
        "difficulty": null
      },
      {
        "id": "adcad0986f7d18faad63643816ec315d861042467f0338a74760f71f9702be8a",
        "domain": "numina",
        "question": "1.2 Mishа noticed that the tram passed by him in 4 seconds, and a tunnel 64 meters long in 12 seconds. Find the length of the tram (in meters), assuming that its speed remains constant throughout the entire observation period.",
        "answer": "32",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5429,
        "difficulty": null
      },
      {
        "id": "47e89d847f55af4f0ddca205aae56775d7d43793af21a08fa05393d812dd9288",
        "domain": "numina",
        "question": "Legyen\n\n$$\n\\begin{aligned}\na+b+c & =0 \\\\\nA+B+C & =0 \\\\\n\\frac{a}{A}+\\frac{b}{B}+\\frac{c}{C} & =0\n\\end{aligned}\n$$\n\nMennyi az\n\n$$\na A^{2}+b B^{2}+c C^{2}\n$$\n\nkifejezés értéke?",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 175814,
        "difficulty": null
      },
      {
        "id": "ea385211f4efd42b0f0f0b1f502001dce6617b6dd4aa1a72c48c205732afe30b",
        "domain": "numina",
        "question": "## Task 5 - 301235\n\nInvestigate whether the sequence $\\left(x_{n}\\right)$ defined by\n\n$$\nx_{1}=1, \\quad x_{n+1}=\\frac{1}{x_{n}+1} \\quad(n=1,2,3, \\ldots)\n$$\n\nis convergent, and determine its limit if it is convergent.",
        "answer": "\\frac{\\sqrt{5}-1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186206,
        "difficulty": null
      },
      {
        "id": "64ec3eb3fc7604f425dbd3f9368621ff8670d0969668b5aeceb95704816bee7e",
        "domain": "numina",
        "question": "1. Write a million: a) using three hundreds and operation signs; b) using six tens and operation signs.",
        "answer": "1000000",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 39271,
        "difficulty": null
      },
      {
        "id": "8133a51577cc0942327cf30c5e560b00dd9991a5b826963f788071975125d098",
        "domain": "numina",
        "question": "Evdokimov M.A.\n\nA convex polyhedron with vertices at the midpoints of the edges of a certain cube is called a cuboctahedron. When a cuboctahedron is intersected by a plane, a regular polygon is obtained. What is the maximum number of sides it can have?",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 67557,
        "difficulty": null
      },
      {
        "id": "4c8a0bf9a461e503d8634e10b4724ff1ef302ceea419e668a89a940e22fea381",
        "domain": "numina",
        "question": "5、A, B, and C each have some apples, with the total being less than 100. If A gives $\\frac{1}{6}$ and $\\frac{1}{4}$ of his apples to B and C respectively, the three will have the same number of apples. Therefore, A originally had the most $\\qquad$ apples, and originally B had  more apples than C. apples.",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138896,
        "difficulty": null
      },
      {
        "id": "30c4ff64039da25080cee359f3c55f3d456b3f030691eb585a8211948aacd302",
        "domain": "numina",
        "question": "## 34. Alsatian Village\n\nThe main street of the village (it is called Main) is perpendicular to General de Gaulle Street; they intersect at the Town Hall Square. The Catholic cathedral faces the main street, while the Protestant one faces General de Gaulle Street. The school is located on the street connecting these two cathedrals, at a bird's flight distance (500 m) from each of the mentioned streets. All streets are straight. One day, the teacher asked his students[^9] to calculate the sum of the reciprocals of the distances (in meters) from the town hall to each of the cathedrals. The students did not know trigonometry or analytic geometry.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_fe999c0fe2ad81fc5164g-061.jpg?height=654&width=1213&top_left_y=455&top_left_x=433)\n\nWhat did they answer their teacher?",
        "answer": "0.002",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 42116,
        "difficulty": null
      },
      {
        "id": "4956d1f644d66c83c6a5e087a1847dc14527a0955be596b71e3280d7d519aef7",
        "domain": "numina",
        "question": "32nd IMO 1991 shortlist Problem 18 What is the largest power of 1991 which divides 1990 m + 1992 n , where m = 1991 1992 and n = 1991 1990 ?",
        "answer": "1991^{1991}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 147883,
        "difficulty": null
      },
      {
        "id": "c16d62a814a49303bfd7d624b409148e7e1e790e39ede559dc86b395a09028e0",
        "domain": "numina",
        "question": "167. To transport 60 tons of cargo from one place to another, a certain number of trucks were required. Due to the poor condition of the road, each truck had to carry 0.5 tons less than originally planned, which is why 4 additional trucks were required. How many trucks were initially required?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36167,
        "difficulty": null
      },
      {
        "id": "bb077d5004fe163bf4f54932369fab17586eaff768a3c459aa416528fbad0e40",
        "domain": "numina",
        "question": "13. (2004 National College Entrance Examination - Jiangsu Paper) Given: $0<\\alpha<\\frac{\\pi}{2}, \\tan \\frac{\\alpha}{2}+\\cot \\frac{\\alpha}{2}=\\frac{5}{2}$. Find the value of $\\sin \\left(\\alpha-\\frac{\\pi}{3}\\right)$.",
        "answer": "\\frac{1}{10}(4-3\\sqrt{3})",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93125,
        "difficulty": null
      },
      {
        "id": "85c726636016a4e3570e4c6d0896c63db382ee0bb3d37f3e663d9853f467bf08",
        "domain": "numina",
        "question": "7. [4] Five guys are eating hamburgers. Each one puts a top half and a bottom half of a hamburger bun on the grill. When the buns are toasted, each guy randomly takes two pieces of bread off of the grill. What is the probability that each guy gets a top half and a bottom half?",
        "answer": "\\frac{8}{63}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151161,
        "difficulty": null
      },
      {
        "id": "a17a527f5bce9e753bd435c1e9c614522846d183b566737dd11e587e20f9482a",
        "domain": "numina",
        "question": "7. Remove the joker cards from a deck of playing cards, and randomly draw 5 cards from the remaining 52 cards. The probability that at least two of the cards have the same number (or letter $J, Q, K, A$) is $\\qquad$ (calculate this probability value, accurate to 0.01).",
        "answer": "0.49",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 116475,
        "difficulty": null
      },
      {
        "id": "d5952c781f525b3f169a3625e97962f1918486e77c704ed0819078149ec1b48e",
        "domain": "numina",
        "question": "17. An old emperor placed his will in an iron box, which was equipped with several locks, each having several keys. The keys were distributed among his five sons. The old emperor required that after his death, the iron box could be opened only if at least three of his sons were present. How many locks are needed at least? And how should the keys be distributed among his sons?",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 125495,
        "difficulty": null
      },
      {
        "id": "87c3deaf66777cc2f0b7934773a154de5eb13a652340c2dd1bb3ee43c397edf8",
        "domain": "numina",
        "question": "[ $\\underline{\\text { Projections of the bases, sides, or vertices of a trapezoid ] }}$ [ Pythagorean Theorem (direct and inverse) ]\n\nThe lateral side of an isosceles trapezoid is 41, the height is 40, and the midline is 45. Find the bases.",
        "answer": "3654",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61240,
        "difficulty": null
      },
      {
        "id": "c749854dde4f2f64062ce41e3816fa4d238f5bc6cc02eb6d0f789084d659548a",
        "domain": "numina",
        "question": "7. Let $P(n)$ denote the product of all non-zero digits of the positive integer $n$, for example: $P(123)=1 \\times 2 \\times 3=6$, $P(206)=2 \\times 6=12$. Then $P(1)+P(2)+\\cdots+P(999)=$ $\\qquad$ —.",
        "answer": "97335",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 111271,
        "difficulty": null
      },
      {
        "id": "0fcba4224e85bc2feec4168a8bba35ad92f8bba04d3f1be4d7388e04d601fb4c",
        "domain": "numina",
        "question": "István exchanged two car lottery tickets worth $5000 \\mathrm{Ft}$ each. On the 20th day of the lottery quarter, he said to Kálmán: \"Take over one of my books. If we win, we will share the prize as interest, in proportion to our stakes and the time elapsed.\" Kálmán gave him 4000 Ft the next day. - A month later, István borrowed 2500 Ft from László under the same conditions. - At the same time, Kálmán said to Miklós: \"Give me 2000 Ft. I have 4000 Ft in a joint car lottery ticket. If it wins, we will split my share as if you had joined me at the same time.\" Miklós accepted the offer.\n\nIn the quarterly draw, both tickets won. One car was appraised at $60000 \\mathrm{Ft}$, the other at $48000 \\mathrm{Ft}$. During the distribution, István wanted to keep the first car. The others acknowledged that they would share the proceeds from the second car. It was only then that Miklós learned that Kálmán's stake had not been in the bet for the entire quarter, which would make their prize smaller. As compensation, Kálmán agreed to calculate Miklós's share as if his money had been in the bet from the beginning of the quarter.\n\nHowever, after the second car was sold, only $42000 \\mathrm{Ft}$ remained for distribution. How much did each of the four participants receive from this?",
        "answer": "2100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 168560,
        "difficulty": null
      },
      {
        "id": "c984d42d85a77302aa4d2d87529257afa282a2855da87b997f133b04e3f32bb0",
        "domain": "numina",
        "question": "Condition of the problem\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\sqrt{\\left(e^{\\sin x}-1\\right) \\cos \\left(\\frac{1}{x}\\right)+4 \\cos x}$",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 45982,
        "difficulty": null
      },
      {
        "id": "cfa1dc89c1bb2aa554261c5a546020dab4062df4b24ca29c06e4e1cf8f1849f4",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=3 p+2 q$\n\n$b=2 p-q$\n\n$|p|=4$\n\n$|q|=3$\n\n$(\\widehat{p, q})=\\frac{3 \\pi}{4}$",
        "answer": "42\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 46327,
        "difficulty": null
      },
      {
        "id": "6c50a8df81e4ccfa7db179b78429325227034f223179ec49a646077c42eeb588",
        "domain": "numina",
        "question": "In a quadrilateral, $a: b: c: d=9: 8: 7: 6$. The angle $\\alpha=72^{\\circ} 36^{\\prime}$ is formed by the sides $a$ and $d$, and the area of the quadrilateral is $225 \\mathrm{~cm}^{2}$. What are the lengths of the sides and the angles of the quadrilateral?",
        "answer": "18.567",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 172093,
        "difficulty": null
      },
      {
        "id": "ea14554c4ef283126144c526cad655053aac56fe1ed5221ce93b6bd72efec2fa",
        "domain": "numina",
        "question": "## SUBJECT I (7p)\n\nConsider the sequence $3,7,11,15,19$,\n\na) Complete the sequence with 3 more terms;\n\nb) Determine the 2014th term;\n\nc) Show that the 2014th term is not a perfect square;\n\nd) Calculate the sum of the first 2014 terms.",
        "answer": "8114406",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179800,
        "difficulty": null
      },
      {
        "id": "717d6eb86f7687f84483b8b95cd054ef4047f8955c312233f37564d1f5c5ba51",
        "domain": "numina",
        "question": "Example 16 (from the 2nd American Mathematical Invitational Competition) The function $f(x)$ is defined on the real number domain and satisfies the following conditions: for any real number $x$, $f(2+x)=f(2-x)$, and $f(7+x)=f(7-x)$. If $x=0$ is a root of the equation $f(x)=0$, how many roots should the equation $f(x)=0$ have at least in the interval $-1000 \\leqslant x \\leqslant 1000$?",
        "answer": "401",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117987,
        "difficulty": null
      },
      {
        "id": "462d42a3a47d4b287c463d3cb62a8d5aad1263ce7b5e1a837cca6d2d2fe07da9",
        "domain": "numina",
        "question": "5. Let $A$ and $B$ be two moving points on the ellipse $x^{2}+3 y^{2}=1$, and $O A \\perp O B$ ($O$ is the origin). Then the product of the maximum and minimum values of $|A B|$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{2\\sqrt{3}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 108812,
        "difficulty": null
      },
      {
        "id": "71b59b1d626feb3a426f1e1a2bd0c1854d61f57a4991ff477595831b68d621f2",
        "domain": "numina",
        "question": "1.1. (2 points) From the digits 1, 3, and 5, different three-digit numbers are formed, each of which has all distinct digits. Find the sum of all such three-digit numbers.",
        "answer": "1998",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 6140,
        "difficulty": null
      },
      {
        "id": "69124596ba0a0672f5a1d7e755a932bf37b7d01afc8386a6f0c9009df0ff1ab0",
        "domain": "numina",
        "question": "2. When a five-digit number is divided by 100, we get a quotient $k$ and a remainder $o$. For how many five-digit numbers is the sum $k+o$ divisible by 11?",
        "answer": "8181",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162608,
        "difficulty": null
      },
      {
        "id": "55b0baa3d9a7e8bfcaae99a0c20edbb7f5bf24bfa5d0c79da39e68b10e8624ac",
        "domain": "numina",
        "question": "11.166. The lengths of the edges of the parallelepiped are $a, b$, and $c$. The edges of lengths $a$ and $b$ are mutually perpendicular, and the edge of length $c$ forms an angle of $60^{\\circ}$ with each of them. Determine the volume of the parallelepiped.",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 49019,
        "difficulty": null
      },
      {
        "id": "74b6bbd2b8459e6ef1147f4cc7350e60710a393f3647a35f71d4218c88f7e5f4",
        "domain": "numina",
        "question": "6. (8 points) As shown in the figure, two circles with a radius of 4 cm are placed as shown, and the two shaded areas in the rectangle are equal. Points $A$ and $B$ are the centers of the two circles. Therefore, the length of $AB$ is $\\qquad$ cm. $(\\pi$ is taken as 3 )\n\n\n\nThe translation is provided as requested, maintaining the original formatting and structure.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 122322,
        "difficulty": null
      },
      {
        "id": "d5f6b78dc1f21376570b0723b76c35d328112af6e5db62cb02db8e84631f9375",
        "domain": "numina",
        "question": "4. How many nonempty subsets of $\\{1,2, \\ldots, 10\\}$ have the property that the sum of its largest element and its smallest element is 11 ?",
        "answer": "341",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154655,
        "difficulty": null
      },
      {
        "id": "a1276bc92ef3d4e5818a91f3dcd57e8fca6e0f9bf1b632489c15f371e82bd36a",
        "domain": "numina",
        "question": "3. Find the value: $\\frac{\\sqrt{3}-4 \\sin 20^{\\circ}+8 \\sin ^{3} 20^{\\circ}}{2 \\sin 20^{\\circ} \\sin 480^{\\circ}}$.",
        "answer": "\\frac{2\\sqrt{3}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117338,
        "difficulty": null
      },
      {
        "id": "96b418e37db9f6227c15632bbb3b95e9221971070e395dadce235696e0e83f2f",
        "domain": "numina",
        "question": "Five, let $\\overline{a b c d}$ be a four-digit number, where the first two digits and the last two digits form two-digit numbers $\\overline{a b}$ and $\\overline{c d}$, respectively. If the following conditions are met: $\\overline{a b} \\neq \\overline{c d}$, $\\overline{a b}$ divides $\\overline{c d}$, and $\\overline{c d}$ divides $\\overline{a b c d}$, then such a four-digit number is called a “lucky number”.\n(1) Find the total number of all “lucky numbers”;\n(2) Find the largest odd “lucky number”.",
        "answer": "65",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 107232,
        "difficulty": null
      },
      {
        "id": "e5428c00e67522e5cdf867dc5bda0a76a6e8dde61fee087eb0d23f18dd9e034b",
        "domain": "numina",
        "question": "4. Find the probability that a number randomly taken from the interval $[0 ; 5]$ is a solution to the equation $\\sin (x+|x-\\pi|)+2 \\sin ^{2}(x-|x|)=0$.\n\n#",
        "answer": "\\frac{\\pi}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 2640,
        "difficulty": null
      },
      {
        "id": "0ab63b707463624b216424a33e638553137e4e49fdc16353fa945779be8b2ef5",
        "domain": "numina",
        "question": "12. B6 (FIN 3) Four distinct circles \\( C, C_{1}, C_{2}, C_{3} \\) and a line \\( L \\) are given in the plane such that \\( C \\) and \\( L \\) are disjoint and each of the circles \\( C_{1}, C_{2}, C_{3} \\) touches the other two, as well as \\( C \\) and \\( L \\). Assuming the radius of \\( C \\) to be 1, determine the distance between its center and \\( L \\).",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 196857,
        "difficulty": null
      },
      {
        "id": "82cc95cce9757aaace02a393f87c29147a68cae3dabffa8d2a61321e340023ed",
        "domain": "numina",
        "question": "Find the number of permutations $\\mathrm{a}_{1}, \\mathrm{a}_{2}, \\ldots, \\mathrm{a}_{10}$ of the numbers $1,2, \\ldots, 10$, such that $\\mathrm{a}_{\\mathrm{i}+1}$ is not less than $\\mathrm{a}_{\\mathrm{i}}-1$ (for $\\mathrm{i}=1,2, \\ldots, 9)$.",
        "answer": "512",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53986,
        "difficulty": null
      },
      {
        "id": "89dbc299de2464be340cea11f20fd696a89cfe6ce01e3722439b1b092c619ea3",
        "domain": "numina",
        "question": "10.017. Given a point $P$, which is 7 cm away from the center of a circle with a radius of $11 \\mathrm{~cm}$. A chord of length 18 cm is drawn through this point. What are the lengths of the segments into which the chord is divided by point $P$?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51141,
        "difficulty": null
      },
      {
        "id": "8af882048f4df71e183a5315bbc1910c11a8ac43583b6a97e641f5a0ce6d8264",
        "domain": "numina",
        "question": "Last year, there were 30 more boys than girls in our scout troop. This year, the number of children in the troop increased by $10 \\%$, while the number of boys increased by $5 \\%$ and the number of girls increased by $20 \\%$.\n\nHow many children do we have in the troop this year?\n\n(L. Hozová)",
        "answer": "99",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76718,
        "difficulty": null
      },
      {
        "id": "055f9817b72067234b1c813cf82e0dc8391e628de5950672d941ab15be5fa2e7",
        "domain": "numina",
        "question": "A diagonal of a regular 2006-gon is called odd if its endpoints divide the boundary into two parts, each composed of an odd number of sides. Sides are also regarded as odd diagonals. Suppose the 2006-gon has been dissected into triangles by 2003 nonintersecting diagonals. Find the maximum possible number of isosceles triangles with two odd sides. (Serbia)",
        "answer": "1003",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 195945,
        "difficulty": null
      },
      {
        "id": "1ad56bb87f326041685bbec4b27e3d97e3da3d690a3475456e4a8b2dec4666a5",
        "domain": "numina",
        "question": "1.1. The Eiffel Tower has a height of 324 m and weighs 10000 tons. How many kilograms will a copy with a height of 1.62 m weigh?",
        "answer": "1.25",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11301,
        "difficulty": null
      },
      {
        "id": "55b88d98ae8d5c59c637e60bc8dde494554571386e3e240593bf40afe6291255",
        "domain": "numina",
        "question": "2.4 In a closed bag lie apples. Three friends tried to lift the bag and guess how many fruits were inside. The first one said that, in his opinion, there were 18 apples, the second thought there were 21, and the third said 22. When the friends opened the bag, it turned out that one of them was off by 2, another by 3, and the third by 6. How many apples were in the bag? Find all possible answers.",
        "answer": "24",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17213,
        "difficulty": null
      },
      {
        "id": "f26efb2fe6cec14aa2d2c10d1e389c475c754b1013b7affd2f51c5368eb4a10e",
        "domain": "numina",
        "question": "5. Given 9 points in space, where no four points are coplanar, line segments are drawn between every pair of points. These line segments can be colored blue or red, or left uncolored. Find the minimum value of $n$ such that, by arbitrarily coloring any $n$ of these line segments with one of the two colors (red or blue), the set of these $n$ line segments will necessarily contain a triangle with all sides of the same color.",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 144054,
        "difficulty": null
      },
      {
        "id": "36502f5109092dc2be7eab23e97867f7f9aa6c2174907fc20ca265096467e207",
        "domain": "numina",
        "question": "## Task Condition\n\nCalculate the limit of the numerical sequence:\n\n$$\n(6-n)^{2}-(6+n)^{2}\n$$",
        "answer": "-\\frac{12}{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 46483,
        "difficulty": null
      },
      {
        "id": "3be540e509b580ef45dc0f47cb297c463d99327a32f627be7aceb1bcc0ae0900",
        "domain": "numina",
        "question": "(9) If the real numbers $x, \\alpha, \\beta$ satisfy $x=\\log _{3} \\tan \\alpha=-\\log _{3} \\tan \\beta$, and $\\alpha-\\beta=$ $\\frac{\\pi}{6}$, then the value of $x$ is $\\qquad$.",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101571,
        "difficulty": null
      },
      {
        "id": "2cc5b59ab4952aed686eeaaa7a5957619ccc6ca0a44b878feb20bf1a6f8dbeb5",
        "domain": "numina",
        "question": "1. Given non-negative real numbers $a_{1}, a_{2}, \\cdots, a_{2022}, b_{1}, b_{2}, \\cdots, b_{2022}, c_{1}, c_{2}, \\cdots, c_{2022}$ satisfy $\\sum_{i=1}^{2022} a_{i}=1$, $\\sum_{i=1}^{2022} b_{i}^{2}=2, \\sum_{i=1}^{2022} c_{i}^{3}=3$. Find the maximum value of $\\sum_{i=1}^{2022} a_{i} b_{i+1} c_{i+2}+\\sum_{i=3}^{2022} a_{i} b_{i-1} c_{i-2}$.",
        "answer": "\\sqrt[3]{12}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 91803,
        "difficulty": null
      },
      {
        "id": "d6ea399aa6c0a801206f5c7612b62e0246b5ba2012ee276afa82eda2ccac7139",
        "domain": "numina",
        "question": "10.357. In trapezoid $ABCD$, the lengths of the bases $AD=24$ cm and $BC=8$ cm, and the diagonals $AC=13$ cm, $BD=5\\sqrt{17}$ cm are known. Calculate the area of the trapezoid.",
        "answer": "80",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 48962,
        "difficulty": null
      },
      {
        "id": "87a24c642f4864034e290923d31da331e2825a5f56f31297d3c9d2e7afe02bd5",
        "domain": "numina",
        "question": "(11) The line $l:(2 m+1) x+(m+1) y-7 m-4=0$ is intersected by the circle $C:(x-1)^{2}+$ $(y-2)^{2}=25$ to form the shortest chord length of $\\qquad$ .",
        "answer": "4\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124307,
        "difficulty": null
      },
      {
        "id": "74f6e4bd279425aa5690aa9462ebd5093c8a8d41a79b0e0904ebb9aaaa447f37",
        "domain": "numina",
        "question": "2. Find the eleventh term of the arithmetic progression if the sum of the first seven terms $S_{7}=77$, and the first term $a_{1}=5$.",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19182,
        "difficulty": null
      },
      {
        "id": "bf73414fa8645305359f14f09d4f930553e9344354f10d58c8814d9f3500dd80",
        "domain": "numina",
        "question": "a) In the figure below, $A D=D C, A E=B D, \\angle A E C=90^{\\circ}$. Determine the value of the angle $\\angle C B D$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-18.jpg?height=463&width=674&top_left_y=2047&top_left_x=588)\nb) In the triangle $\\triangle A B C$ below, $B D$ is the bisector of the angle $\\angle A B C, E$ is the midpoint of $A C$, and $\\angle A D B=90^{\\circ}$. If $A B=12 \\mathrm{~cm}$ and $B C=20 \\mathrm{~cm}$, calculate the length of the segment $D E$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-19.jpg?height=534&width=990&top_left_y=424&top_left_x=630)",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73531,
        "difficulty": null
      },
      {
        "id": "2315864c5cdde3b7087111c9b562d6e6adfc228961769fa9b72714240d57c426",
        "domain": "numina",
        "question": "## Task 1 - 100811\n\nDetermine the number of all six-digit natural numbers in which the digit sequence 1970 (i.e., the base digits 1, 9, 7, 0 in this order and without any other digits in between) occurs!\n\nWhat is the smallest and what is the largest of these six-digit numbers?",
        "answer": "280",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 184031,
        "difficulty": null
      },
      {
        "id": "272cace911d2f4d3ce849dfbe82ffe1a98b137bedd4fe4c44a0ec469b496a108",
        "domain": "numina",
        "question": "5、Let $a, b, c (c > a, b)$ be the lengths of the three sides of a right triangle, and the point $(m, n)$ lies on the line $a x + b y + c = 0$. Then the minimum value of $m^{2} + n^{2}$ is $\\qquad$ .",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 132361,
        "difficulty": null
      },
      {
        "id": "db6fbab438efca1aeed0c274d1360185bcd9d13ff2d467941a6b6a183e0cc3a3",
        "domain": "numina",
        "question": "10. For the geometric sequence $\\left\\{a_{n}\\right\\}$ with the first term $a_{1}=1536$, common ratio $q=-\\frac{1}{2}$, $\\Pi_{n}$ represents the product of its first $n$ terms $\\left(n \\in \\mathbf{N}^{*}\\right)$, then the value of $n$ that maximizes $\\mathrm{I}_{n}$ is $\\qquad$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141839,
        "difficulty": null
      },
      {
        "id": "f37c67071518cbb50f13caf80644ce620879eac36dc488ec20723e24ed8e8398",
        "domain": "numina",
        "question": "Let's determine the minimum and maximum values of the expression $9-x^{2}-2 \\sqrt{9-x^{2}}$, where $x$ is a real number between -3 and 3.",
        "answer": "-13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171611,
        "difficulty": null
      },
      {
        "id": "d7f68e68542125251456e7d00354607b2738cc7f940190f39430780ecfc64a8c",
        "domain": "numina",
        "question": "## Task B-2.3.\n\nIn online shopping, a package of photocopy paper is 9 kuna cheaper than in a bookstore. In December, the secretary bought 8 more packages online than she bought in November for the same amount of 1440 kn in the bookstore. What is the price of photocopy paper in online shopping?",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88725,
        "difficulty": null
      },
      {
        "id": "cec6fd0fa536ea0336d7798f0b846ed23ff8bd017668eecb717c1e2d5e84c501",
        "domain": "numina",
        "question": "2. Both rows of the following grid have the same sum. What is the value of *?\n\\begin{tabular}{|r|r|r|r|r|r|r|r|r|r|c|}\n\\hline 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 & 10 & 1050 \\\\\n\\hline 11 & 12 & 13 & 14 & 15 & 16 & 17 & 18 & 19 & 20 & $*$ \\\\\n\\hline\n\\end{tabular}",
        "answer": "950",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156420,
        "difficulty": null
      },
      {
        "id": "aef811313cc856e99736bef967ce93e512dc42495e54e37a4883497566ec945c",
        "domain": "numina",
        "question": "10. (3 points) The greatest common divisor and the least common multiple of two numbers are 3 and 135, respectively. Find the minimum difference between these two numbers is $\\qquad$ .",
        "answer": "12",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 107185,
        "difficulty": null
      },
      {
        "id": "fe7fa3a98f84c1068405a1443512356ffac488666720151b3eb7b45795a4f953",
        "domain": "numina",
        "question": "4. Find the smallest number $n>4$ for which there exists a set of $n$ people such that any two who know each other have no common acquaintances, and any two who do not know each other have exactly two common acquaintances. (Acquaintance is a symmetric relation: if $A$ knows $B, A \\neq B$, then $B$ also knows $A$.)\n\n(Bulgarian)",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 161393,
        "difficulty": null
      },
      {
        "id": "f84f6eed9230210a6c1edb464f24db915635f6d129ade8c5566c07eca8cb6002",
        "domain": "numina",
        "question": "1. For a finite set $A$, there exists a function $f: \\mathbf{N}_{+} \\rightarrow A$, with the following property: if $i, j \\in \\mathbf{N}_{+}$ and $|i-j|$ is a prime number, then $f(i) \\neq f(j)$. How many elements does the set $A$ have at minimum?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 108129,
        "difficulty": null
      },
      {
        "id": "ae216d702996fbf5a2b3f5818f45157500aca2dcc969762a2dc9ba606e53e260",
        "domain": "numina",
        "question": "7. Find the largest integer which cannot be expressed as sum of some of the numbers $135,136,137, \\ldots, 144$ (each number may occur many times in the sum or not at all).\n(1 mark)\n某整數不能寫成 135、136、‥144 當中的某些數之和（在和式中每個數出現的次數不限, 亦可以不出現）。求該整數的最大可能值。",
        "answer": "2024",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153617,
        "difficulty": null
      },
      {
        "id": "61008854a84844c30c5a5f7eb273fdf040ec2894b55a3dfd0226e7a6fc7f58a6",
        "domain": "numina",
        "question": "10. To pack books when moving a school library, you can buy small boxes that hold 12 books or large ones that hold 25 books. If all the books are packed in small boxes, 7 books will be left, and if all the books are packed in large boxes, there will be room for 5 more books. The library's collection contains between 500 and 650 books. How many books are in the library?",
        "answer": "595",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 19218,
        "difficulty": null
      },
      {
        "id": "97f2ac34670b6a0513f36520cebcfdcda38728aab6a1d7e9d1f8d240c5380d63",
        "domain": "numina",
        "question": "2. Let $f(x)=\\min (4 x+1, x+2,-2 x+4)$, then the maximum value of $f(x)$ is $\\qquad$",
        "answer": "\\frac{8}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97210,
        "difficulty": null
      },
      {
        "id": "867ab8a49aa5e91e656e08f5e9c08987a551ce04c337b589554653d668c991d2",
        "domain": "numina",
        "question": "5. (5 points) If 6 is placed at the right end of a two-digit number, the resulting three-digit number is 474 greater than the original two-digit number. The original two-digit number is $\\qquad$ .",
        "answer": "52",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98136,
        "difficulty": null
      },
      {
        "id": "70f8a03d6f215616b481d65924f1a5344be318f881ef22bd078114c7f8c955c4",
        "domain": "numina",
        "question": "4. Given are the numbers $a=-8$ and $b=36$.\n\na) Calculate the value of the expression $||a|-2 \\sqrt{b}|-\\frac{1}{2}|\\sqrt[3]{a}|$.\n\nb) Solve the equation $|x-\\sqrt[3]{a}|=\\sqrt{b}$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163621,
        "difficulty": null
      },
      {
        "id": "23b946d84cd5d859b5aa45d02bda5eec42e7addd0f7d3b8ddbe0479251158dd6",
        "domain": "numina",
        "question": "83. What is in the period? Determine what stands in the period of the decimal expansion of the fraction $\\frac{1}{49}$.[^3]",
        "answer": "0.020408163265306122448979591836734693877551",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41516,
        "difficulty": null
      },
      {
        "id": "21eddb43787bfa4b9a83e618c02717ec2110aec608706747d6c143c1ee119afa",
        "domain": "numina",
        "question": "\\section*{Problem 6 - 320936}\n\na) Give three integers \\(x, y\\) and \\(z\\) such that:\n\n\\[\nx^{2}+y^{2}+z^{2}-4 x+12 y-14 z-57=0\n\\]\n\nb) Determine the number of all triples \\((x, y, z)\\) of integers \\(x, y, z\\) that satisfy equation (1)!",
        "answer": "192",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 184775,
        "difficulty": null
      },
      {
        "id": "efaf189c7af4ffce6248a894d41609fa0156ca58a35436a64374d3c0aba303cf",
        "domain": "numina",
        "question": "ALB\n\n3) If $x^{3}-3 \\sqrt{3} x^{2}+9 x-3 \\sqrt{3}-64=0$, find the value of $x^{6}-8 x^{5}+13 x^{4}-5 x^{3}+49 x^{2}-137 x+2015$.",
        "answer": "1898",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 197766,
        "difficulty": null
      },
      {
        "id": "db41328407b1604e2e41476421b4b8805113797977c113983420e18737dd83cc",
        "domain": "numina",
        "question": "Consider the positive integer\n\n$$\nn=2^{r}-16^{s}\n$$\n\nwhere $r$ and $s$ are also positive integers. Find the conditions that $r$ and $s$ must satisfy for the remainder of the division of $n$ by 7 to be 5. Find the smallest number that satisfies this condition.\n\n## Solution.",
        "answer": "768",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86239,
        "difficulty": null
      },
      {
        "id": "f53db7e0af83feb814ca3df655b2454e91843a63905ebef42e1608514a7f20c0",
        "domain": "numina",
        "question": "2. Find the smallest prime number $p$ for which the number $p^{3}+2 p^{2}+p$ has exactly 42 positive divisors.",
        "answer": "23",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 162683,
        "difficulty": null
      },
      {
        "id": "95362216df2d0006fb52c7b764ea91777c49cf61bcc53d20e56e6a608f65d350",
        "domain": "numina",
        "question": "3. Let $v(X)$ be the sum of elements of a nonempty finite set $X$, where $X$ is a set of numbers. Calculate the sum of all numbers $v(X)$ where $X$ ranges over all nonempty subsets of the set $\\{1,2,3, \\ldots, 16\\}$.",
        "answer": "4456448",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154253,
        "difficulty": null
      },
      {
        "id": "80d84961420311253d385ccb8d064b96345937e6c0a8e7954906b148d0face8d",
        "domain": "numina",
        "question": "3. From points $A$ and $B$, two cars set off towards each other simultaneously with constant speeds. One hour before the first car arrived at $B$ and four hours before the second car arrived at $A$, they met. Find the ratio of the speeds of the cars.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14667,
        "difficulty": null
      },
      {
        "id": "8359d5dbcaf63432bcb7b83c02e9847ae1f1e04918efa2e67f1a03c7be67a46a",
        "domain": "numina",
        "question": "## Task 2 - 340812\n\nA three-digit natural number is called \"symmetric\" if and only if its hundreds digit is equal to its units digit.\n\na) Form all those three-digit symmetric numbers in which only the digits $0,1,2$ appear (each possibly multiple times or not at all)! No justification is required.\n\nb) Form all those three-digit symmetric numbers that are divisible by 6 and in which only the digits 2, 5, 8 appear! Prove that exactly the numbers you specify meet the given conditions!\n\nc) Determine the number of all three-digit symmetric numbers!\n\nd) Determine the sum of all three-digit symmetric numbers!",
        "answer": "49500",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 184319,
        "difficulty": null
      },
      {
        "id": "e713d996164816607c1b6e5171eb03ee390d3328efe4f7ee4a13edfc700741c6",
        "domain": "numina",
        "question": "5. Solution. First, let's prove that building towers taller than 2 is meaningless.\n\nIndeed, suppose one of the towers has a height of \\( h > 2 \\). Let's remove the top floor from it, turning it into a new tower of height 1. How will this change the inspector's sum? The shortened tower no longer sees towers of height \\( h-1 \\). However, since \\( h > 2 \\), from all these towers (including the shortened one), the new tower is now visible. Therefore, the total sum has increased.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4ca17689025243fab126g-18.jpg?height=319&width=616&top_left_y=1354&top_left_x=465)\n\nFig. 15\n\nIt remains to study the case where all towers have a height of 1 or 2. If there are \\( X \\) towers of height 1 and \\( Y \\) towers of height 2 (by the condition \\( X + 2Y = 30 \\)), then the inspector's sum is\n\n\\[\nX Y = (30 - 2Y) Y = 2Y(15 - Y) = 2\\left(7.5^2 - (Y - 7.5)^2\\right)\n\\]\n\nFor integer \\( Y \\), this expression takes its maximum value at \\( Y = 7 \\) and \\( Y = 8 \\), and equals 112.",
        "answer": "112",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 23297,
        "difficulty": null
      },
      {
        "id": "c9201c7c0abf26771be0e226611fc91939a18ecb3b5a66920679034f30683f6c",
        "domain": "numina",
        "question": "## Task B-4.4.\n\nHow many passwords consisting of 4 characters from the set $\\{a, b, c, 1,2,3,4,5,+,!\\}$ are there such that all characters in the password are different or the password contains exactly two, not necessarily different letters and two, not necessarily different digits?",
        "answer": "5670",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88691,
        "difficulty": null
      },
      {
        "id": "7b3135b5e1c5118f58192ea278ac666cafc58f5690ddb0a338d070ec32ab6c36",
        "domain": "numina",
        "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-10.5,10.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
        "answer": "462",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19693,
        "difficulty": null
      },
      {
        "id": "7a4e319cd6309c8acd038c3ed5dd379f1a442373cd1fe2a5aaeaeaeffaae4c77",
        "domain": "numina",
        "question": "Circle inscribed in an angle  Area of a trapezoid\n\nA circle with a radius of 3 is inscribed in a right trapezoid, the smaller base of which is 4. Find the larger base of the trapezoid.",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64765,
        "difficulty": null
      },
      {
        "id": "421dc4c845cbc19a51e7218c1d41609ccf7d0d94b6e5e31317b25dac4b603e3e",
        "domain": "numina",
        "question": "1. (5 points) Find the degree measure of the angle\n\n$$\n\\delta=\\arccos \\left(\\left(\\sin 2195^{\\circ}+\\sin 2196^{\\circ}+\\cdots+\\sin 5795^{\\circ}\\right)^{\\cos } 2160^{\\circ}+\\cos 2161^{\\circ}+\\cdots+\\cos 5760^{\\circ}\\right)\n$$",
        "answer": "55",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19575,
        "difficulty": null
      },
      {
        "id": "1871c6301f052c1a3e8eea52ed53dc3fde269385083ccb881e9527266ab94fce",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n$\\lim _{x \\rightarrow 0}\\left(\\frac{2+x}{3-x}\\right)^{x}$",
        "answer": "1",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 45688,
        "difficulty": null
      },
      {
        "id": "65151a0f390f4e5a3e233f6280aeee41af8eae5b5019f9671ae9d999be9e45a0",
        "domain": "numina",
        "question": "3. Given is the quadratic equation in $x$\n\n$$\n4(p-2)^{2} x^{2}-4(p-2) q x+q(4 p-3 q+4)-(p-1)^{2}=0, p, q \\in \\mathbb{N}\n$$\n\nWhat condition must $p$ and $q$ satisfy for the equation to have a double root? Find at least one double root that is a natural number.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190734,
        "difficulty": null
      },
      {
        "id": "2aa0f64589e0ecc7c5a4e2823d27c78dc2cb3e2083aa544b4b3dc73627e07b2d",
        "domain": "numina",
        "question": "Example 6.25. A clerk accepts on average two applications per hour. It is assumed that the flow of applications is a simple one. What is the probability of receiving four applications in four hours?",
        "answer": "0.9576",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 30411,
        "difficulty": null
      },
      {
        "id": "d7bd90a2fe4f672a0ae8e4df5983331a0e8968e3d0e1fa2706a2caebbffc541f",
        "domain": "numina",
        "question": "2. (10 points) Calculate:\n$$\n10 \\times 9 \\times 8 + 7 \\times 6 \\times 5 + 6 \\times 5 \\times 4 + 3 \\times 2 \\times 1 - 9 \\times 8 \\times 7 - 8 \\times 7 \\times 6 - 5 \\times 4 \\times 3 - 4 \\times 3 \\times 2\n$$",
        "answer": "132",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 137616,
        "difficulty": null
      },
      {
        "id": "6dee0d576d1c6bc5751a942e4557c96477e29774a9789aea3895a15a35ea414b",
        "domain": "numina",
        "question": "1. Class A and Class B have a total of 83 people, Class B and Class C have a total of 86 people, Class C and Class D have a total of 88 people. How many people are there in Class A and Class D?",
        "answer": "85",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107152,
        "difficulty": null
      },
      {
        "id": "e69bd6366c1f81591462c158b4592dc393d488f12a3199c7945c31a4e8996f78",
        "domain": "numina",
        "question": "8. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=$ $A D=5, A A_{1}=1, M$ is a point on edge $A B$ and satisfies $3 \\overrightarrow{A M}+2 \\overrightarrow{B M}=\\mathbf{0}$. A point $P$ is randomly selected within the square base $A B C D$, and the distance from point $P$ to line $A_{1} D_{1}$ is $d_{1}, P M=d_{2}$. Then the probability that the point $P$ satisfies $d_{1}^{2}-d_{2}^{2} \\geqslant 1$ is $\\qquad$",
        "answer": "\\frac{32}{75}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114447,
        "difficulty": null
      },
      {
        "id": "a2e7e32d5e0c83520e5838940eab1452da31064467d2d568056c7154046854b3",
        "domain": "numina",
        "question": "12. What is the value of $\\sqrt{20212020 \\times 20202021-20212021 \\times 20202020}$ ?",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 156382,
        "difficulty": null
      },
      {
        "id": "e6f2eeb3a2ec3ec3088f9843a77c9bbac826dfeec531bb5f3275d5f6304f0dbe",
        "domain": "numina",
        "question": "50*. What is the greatest number of circles of radius 1 that can be placed on a plane so that they all intersect a certain fixed unit circle $S$ and no one of them contains the center of $S$ or the center of another circle inside it?",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 28312,
        "difficulty": null
      },
      {
        "id": "d892af3aa7cc5917e18f74b16e638e9a3b9caa58840b1d01bd03c267d9a71ee3",
        "domain": "numina",
        "question": "1. Given two quadratic trinomials $f(x)$ and $g(x)$ with leading coefficients of 1, each having two real roots, and $f(1)=g(2)$, $g(1)=f(2)$. Find the sum of the four roots of these two quadratic trinomials.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105903,
        "difficulty": null
      },
      {
        "id": "c4995db98f60ffe3ba1bb5b92d24055c358a5af39815df90eb6b075f7a1789a0",
        "domain": "numina",
        "question": "Example 19 (2002 National High School Competition Question) As shown in Figure 5-19, there is a sequence of curves $P_{0}, P_{1}, P_{2}, \\cdots$. It is known that the figure enclosed by $P_{0}$ is an equilateral triangle with an area of 1. $P_{k+1}$ is obtained by performing the following operation on $P_{k}$: trisect each side of $P_{k}$, construct an equilateral triangle outward using the middle segment of each side, and then remove the middle segment $(k=0,1,2, \\cdots)$. Let $S_{n}$ be the area of the figure enclosed by the curve $P_{n}$.\n(1) Find the general term formula for the sequence $\\left\\{S_{n}\\right\\}$;\n(2) Find $\\lim _{n \\rightarrow \\infty} S_{n}$.",
        "answer": "\\frac{8}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 115606,
        "difficulty": null
      },
      {
        "id": "8cf5fb9ab4f8837fb4956a90fc4b7abc0910415c547570fe71c37a5a1283096b",
        "domain": "numina",
        "question": "## Task Condition\n\nFind the angle between the planes:\n\n$x+4 y-z+1=0$\n\n$2 x+y+4 z-3=0$",
        "answer": "845^{\\}4^{\\\\}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 46267,
        "difficulty": null
      },
      {
        "id": "fcfa0ab70fa7bac28a71ec4170d647a70f26e77e7deca99bab0caf1dc1ad149e",
        "domain": "numina",
        "question": "Water can be let into a basin through four different taps. If the first and second taps are opened simultaneously, the basin fills up in 2 hours. If the second and third taps are opened at the same time, it takes 3 hours, and if the third and fourth taps are opened simultaneously, it takes 4 hours for the basin to fill up. How long will it take to fill the basin if water flows in through the fourth and first taps at the same time?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170392,
        "difficulty": null
      },
      {
        "id": "90cf4605e60c04b615b3e6cea9f795b480c3340512b7713c3528c961f403063e",
        "domain": "numina",
        "question": "8. Let set $S$ consist of all positive integers not exceeding $2^{40}$, and whose binary representation contains exactly two 1s. If a number is randomly selected from $S$, the probability that this number is divisible by 9 is $\\qquad$ .",
        "answer": "\\frac{133}{780}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144159,
        "difficulty": null
      },
      {
        "id": "6e9271d19b93ae6912139da65c6f27796003a104f4cc905668d5fdca6b722fbe",
        "domain": "numina",
        "question": "2. Calculate the value of the expression\n\n$$\n\\left(\\frac{-1+i \\sqrt{3}}{2}\\right)^{1986}+\\left(\\frac{-1-i \\sqrt{3}}{2}\\right)^{1986}\n$$\n\ni is the imaginary unit.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190733,
        "difficulty": null
      },
      {
        "id": "dca42ad022a773b08fb8930dc7bf0dde0d92b33ac2f906e4b47c3bb227e56eaa",
        "domain": "numina",
        "question": "A $180 \\mathrm{~cm}$ high, horizontally based, cylindrical fuel tank is equipped with a tap at the bottom and a filling nozzle at the top. The latter can fill the tank in 1 hour. If we open both the filling nozzle and the tap for 5 minutes, the fuel level in the tank rises by $10 \\mathrm{~cm}$. How long will it take to empty a full tank through the tap?",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 174583,
        "difficulty": null
      },
      {
        "id": "06939d7f777d58f100443d006e0cfca1babb6e4f195d039576200a8859b74895",
        "domain": "numina",
        "question": "251. Packing Cylinders. Forty cylinders with a diameter of 1 cm and the same height were tightly packed in a box in 5 rows of 8 cylinders each so that they would not \"rattle\" during transportation. How many cylinders need to be removed from the box so that, by moving the remaining cylinders and adding the removed cylinders and one more cylinder at the end, 41 cylinders of the same size can be packed in this box? Will the cylinders \"rattle\" in this case?\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_2b40d984baf9da275c68g-054.jpg?height=325&width=512&top_left_y=1448&top_left_x=772)",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 41684,
        "difficulty": null
      },
      {
        "id": "62c60e23426c2f7dd3964d66fdb147898ccb548a21f89063199cec15a390ee1e",
        "domain": "numina",
        "question": "13th CanMO 1981 Problem 2 The circle C has radius 1 and touches the line L at P. The point X lies on C and Y is the foot of the perpendicular from X to L. Find the maximum possible value of area PXY (as X varies). Solution",
        "answer": "\\frac{3\\sqrt{3}}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 149587,
        "difficulty": null
      },
      {
        "id": "6a3c041a3b102fa393f868a499bafdcfe055564c226b8d1e2d83b52e4123a233",
        "domain": "numina",
        "question": "15. Find the sum:\n\n$$\n1+2-3-4+5+6-7-8+\\ldots+301+302\n$$",
        "answer": "303",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39592,
        "difficulty": null
      },
      {
        "id": "d7a01af304a43d2bf6c887df5279cc3110d2d92a29159e7329e18223a76aa53e",
        "domain": "numina",
        "question": "(not treated in class) Determine the minimum of $x^{2}+y^{2}+z^{2}$ under the condition $x^{3}+y^{3}+z^{3}-3 x y z=1$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 82304,
        "difficulty": null
      },
      {
        "id": "5993584729e4a8008e751a10e09a3272a0632c9ee1a57476397ff7370f8a15e8",
        "domain": "numina",
        "question": "18. Prince Gvidon had three sons. Among his descendants, 93 each had two sons and no daughters, while all the others died childless. How many descendants did Prince Gvidon have in total?",
        "answer": "189",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 30052,
        "difficulty": null
      },
      {
        "id": "fc2bceb7d6f4eb8d0590f9e0d1526a74c1bf59523e7d0df69e51d57665208879",
        "domain": "numina",
        "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow \\frac{\\pi}{2}} \\frac{e^{\\tan 2 x}-e^{-\\sin 2 x}}{\\sin x-1}$",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46495,
        "difficulty": null
      },
      {
        "id": "7e838f1981946b3a938b7a167d9e339902a6c8a690caa3dc51d189ae9a73a5af",
        "domain": "numina",
        "question": "Task B-4.2. For what values of the real number $x$ in the expansion of $\\left(\\sqrt{5^{x}}+\\frac{1}{\\sqrt[3]{25^{x}}}\\right)^{6}$ is the third term equal to 75?",
        "answer": "-\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88901,
        "difficulty": null
      },
      {
        "id": "799a78c42c8e45fb23ca64dd159ab98dcbdb970a8e235c98c367fa3a9e08eb84",
        "domain": "numina",
        "question": "78. The smallest natural number whose digits sum to 218 is an $n$-digit number, then $n=$",
        "answer": "25",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122665,
        "difficulty": null
      },
      {
        "id": "9f67e537f4678c19cff29a68d899e2f82b0e7c781030c33b33bdb7b7c011adde",
        "domain": "numina",
        "question": "B4. Calculate the integer solution of the equation $\\log _{2}\\left(x^{2}+7\\right)=\\frac{\\log _{2}(x+1)}{\\log _{8} 2}-\\log _{2} x$. (7 points)\n\nSpace for solving problems in set $B$.\n\n## dIFA\n\n12th Mathematics Knowledge Competition for Students of Secondary Technical and Vocational Schools\n\nRegional Competition, March 28, 2012\n\n## PROBLEMS FOR THE FOURTH YEAR\n\nTime for solving: 90 minutes. In set A, we will evaluate a correct answer with two points, while we will deduct half a point for an incorrect answer. Write the answers for set A in the left table.\n![](https://cdn.mathpix.com/cropped/2024_06_07_d6b556bd14f8001ca6fag-08.jpg?height=172&width=1210&top_left_y=658&top_left_x=433)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 164224,
        "difficulty": null
      },
      {
        "id": "299e1a9d21ed2f63869d48962102e16d3dc246b0a7499fe63bf7294048aed226",
        "domain": "numina",
        "question": "4. 42 Find the positive root of the equation $x=\\sqrt{2+\\sqrt{2+\\sqrt{2+\\sqrt{2+x}}}}$, and prove that there is only one positive root.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99884,
        "difficulty": null
      },
      {
        "id": "03563142e473150e6135b9bb7b5071f2d8e69c174afbdc284a8fc7c5d63255eb",
        "domain": "numina",
        "question": "5. In the number $2 * 0 * 1 * 6 * 0 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,1,2,3,4,5,6,7,8$ (digits can repeat) so that the resulting 10-digit number is divisible by 18. In how many ways can this be done?",
        "answer": "3645",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 3597,
        "difficulty": null
      },
      {
        "id": "856cc6322cf64dddea596c6cf356290be84c3cf1f0b951d430f976c01ce4cacf",
        "domain": "numina",
        "question": "11. (10 points) Several people completed the task of planting 2013 trees, with each person planting the same number of trees. If 5 people did not participate in planting, the rest would need to plant 2 more trees each to fail to complete the task, while planting 3 more trees each would exceed the task. How many people participated in planting?",
        "answer": "61",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131117,
        "difficulty": null
      },
      {
        "id": "63a105abe8de002016fdbceaedc93e12816ff55bf3de202f28a0e8e3f978a043",
        "domain": "numina",
        "question": "3. A road 1 km long is fully illuminated by street lamps, each of which illuminates a section of the road 1 m long. What is the maximum number of street lamps that can be on the road, given that after turning off any street lamp, the road will no longer be fully illuminated?",
        "answer": "1998",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 25863,
        "difficulty": null
      },
      {
        "id": "43b30f8198c75d856e2a9e596643a22703733c569b1f7df36323f61f7f6c6ccd",
        "domain": "numina",
        "question": "6.164. $\\sqrt{x+8+2 \\sqrt{x+7}}+\\sqrt{x+1-\\sqrt{x+7}}=4$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 48235,
        "difficulty": null
      },
      {
        "id": "652cd3e336bd9c872ab8012e19311ae04ac5bffc471de505779df843df40b657",
        "domain": "numina",
        "question": "10. (20 points) Given a line $l$ intersects the ellipse $C: \\frac{x^{2}}{2}+y^{2}=1$ at points $A, B$, and $O$ is the origin.\n(1) Find the maximum value of the area of $\\triangle A O B$, and the equation of the line $l$ at this time;\n(2) Let the set $L=\\{l \\mid$ line $l$ makes the area of $\\triangle A O B$ take the maximum value $\\}, l_{1}, l_{2}, l_{3}, l_{4} \\in L$ and satisfy $l_{1} / / l_{2}, l_{3} / / l_{4}$, $k_{1}, k_{2}, k_{3}, k_{4}$ are the slopes of $l_{1}, l_{2}, l_{3}, l_{4}$ respectively and satisfy $k_{1}+k_{2}+k_{3}+k_{4}=0$, find the minimum value of the area of the quadrilateral formed by these four lines.",
        "answer": "2\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105959,
        "difficulty": null
      },
      {
        "id": "e69b0d487e73369b2bf0228b7e3f76bc458edf93276a8e4059795cece11d586a",
        "domain": "numina",
        "question": "1. Let $A B C$ be a triangle such that $A B=7$, and let the angle bisector of $\\angle B A C$ intersect line $B C$ at $D$. If there exist points $E$ and $F$ on sides $A C$ and $B C$, respectively, such that lines $A D$ and $E F$ are parallel and divide triangle $A B C$ into three parts of equal area, determine the number of possible integral values for $B C$.",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151313,
        "difficulty": null
      },
      {
        "id": "841d7e31cfb2f1e0531e56fb3645674abe984f8d45c24ec48933f3a693d32586",
        "domain": "numina",
        "question": "4. Consider the adjacent complementary angles $\\square A O B$ and $\\square B O C$, points $D, E \\in \\operatorname{Int}(\\square A O B)$, such that $\\square A O D \\equiv \\square D O E \\equiv \\square E O B$, and (OF is the bisector of angle $\\square B O C$. Given that $\\mathrm{m}(\\square E O F)=39^{\\circ}$, determine:\n\n(4p) a) $\\mathrm{m}(\\square A O B)$ and $\\mathrm{m}(\\square B O C)$.\n\n(3p) b) m( $\\square D O M$ ), where ( $O M$ is the bisector of angle $\\square E^{\\prime} O C$, and ( $O E$ and ( $O E^{\\prime}$ are opposite rays.\n\nAdina Oancea\n\nNote: All subjects are mandatory.\n\nEffective working time: 2 hours.",
        "answer": "135",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 178111,
        "difficulty": null
      },
      {
        "id": "5ca33c1b0f52d6d0dce2c377ca16f7d79e6253aa20211cfbfbcf8438582ad3d8",
        "domain": "numina",
        "question": "Three. (50 points) Nine small balls numbered $1,2, \\cdots, 9$ are randomly placed on nine equally spaced points on a circle, with one ball on each point. Let $S$ be the sum of the absolute values of the differences between the numbers of all adjacent balls on the circle. Find the probability of the arrangement that minimizes $S$.",
        "answer": "\\frac{1}{315}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 133798,
        "difficulty": null
      },
      {
        "id": "86d01e41d79b461d03465dfc76205934de1d51c4332afe25b3e0d40b1fd997ae",
        "domain": "numina",
        "question": "Problem 3. If $x y=6$ and $x^{2} y+x y^{2}+x+y=63$, then what is $x^{2}+y^{2}$?",
        "answer": "69",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188549,
        "difficulty": null
      },
      {
        "id": "c289468867b5fed8ac4f7c62164ae919ba970b79ce45b9d1e0874f8578c4d3b7",
        "domain": "numina",
        "question": "5. (40th IMO Problem) $n$ is a given integer, $n \\geqslant 2$, determine the smallest constant $c$, such that the inequality $\\sum_{1 \\leq i \\leq j \\leq n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4}$ holds for all non-negative real numbers $x_{1}, x_{2}, \\cdots, x_{n}$.",
        "answer": "\\frac{1}{8}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 120902,
        "difficulty": null
      },
      {
        "id": "9d4f236c0eed322c64d7c13612942ced3896807f10341c8b3dd167080cef9cbf",
        "domain": "numina",
        "question": "2、The following is a calendar for May of a certain year. Using a $2 \\times 2$ box that can frame four numbers (excluding Chinese characters), the number of different ways to frame four numbers is $\\qquad$.\n\n---\n\nNote: The original text includes a placeholder for the answer (indicated by $\\qquad$). This has been preserved in the translation.",
        "answer": "20",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 132163,
        "difficulty": null
      },
      {
        "id": "ca15e970fe4f4c23d1596768dd70c683484c172c39b285e8ca7228e16ee6d795",
        "domain": "numina",
        "question": "27. If the team had scored 97 points in the last sport, the average number of points scored by it in one sport would have been 90; if it had scored 73 points in this last sport, the average number of points per sport would have been 87. In how many sports did the team compete?",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39636,
        "difficulty": null
      },
      {
        "id": "5bb537d6de60cbb0057309c430f3c08670925b839693c14ca00828cae069bed9",
        "domain": "numina",
        "question": "Problem 4.3. How many rectangles exist on this picture with sides running along the grid lines? (A square is also a rectangle.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_038330058549e9b1ba1cg-06.jpg?height=173&width=206&top_left_y=614&top_left_x=624)",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 14280,
        "difficulty": null
      },
      {
        "id": "5805019a5884e92c8a3384e96daf5263cc571b0127bb938a12395e7ef4b1b1db",
        "domain": "numina",
        "question": "36. [20] In the game of Connect Four, there are seven vertical columns which have spaces for six tokens. These form a $7 \\times 6$ grid of spaces. Two players White and Black move alternately. A player takes a turn by picking a column which is not already full and dropping a token of their color into the lowest unoccupied space in that column. The game ends when there are four consecutive tokens of the same color in a line, either horizontally, vertically, or diagonally. The player who has four tokens in a row of their color wins.\nAssume two players play this game randomly. Each player, on their turn, picks a random column which is not full and drops a token of their color into that column. This happens until one player wins or all of the columns are filled. Let $P$ be the probability that all of the columns are filled without any player obtaining four tokens in a row of their color. Estimate $P$.\nAn estimate of $E>0$ earns $\\lfloor 20 \\min (P / E, E / P)\\rfloor$ points.",
        "answer": "0.0025632817",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151710,
        "difficulty": null
      },
      {
        "id": "ca40ddb386d2d0bf24e49ef37ec1bdb5a381cb18fb2a4897e35144e00aa99c11",
        "domain": "numina",
        "question": "5.1. Let $S(n)$ be the sum of the digits in the decimal representation of the number $n$. Find $S\\left(S\\left(S\\left(S\\left(2017^{2017}\\right)\\right)\\right)\\right)$. Answer. 1.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6288,
        "difficulty": null
      },
      {
        "id": "679b23fa45bddd2f4254dc6d1e6a164138028710edfd535f630d9e822ac23f40",
        "domain": "numina",
        "question": "Anzhans A.\n\nThere are 100 silver coins, ordered by weight, and 101 gold coins, also ordered by weight. It is known that all coins have different weights. We have a two-pan balance that allows us to determine which of any two coins is heavier. How can we find the coin that ranks 101st in weight among all the coins with the fewest number of weighings?",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56379,
        "difficulty": null
      },
      {
        "id": "51972e1e341463bc94462440c5682054e1adc67cfe9dcc636418985124047803",
        "domain": "numina",
        "question": "Let $\\left(a_{1}, a_{2}, \\ldots, a_{100}\\right)$ be a permutation of $(1,2, \\ldots, 100)$.\n\nWhat is the maximum number of perfect squares that can occur among the numbers $a_{1}, a_{1}+a_{2}, a_{1}+a_{2}+a_{3}, \\ldots, a_{1}+\\ldots+a_{100} ?$",
        "answer": "60",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 84902,
        "difficulty": null
      },
      {
        "id": "f2ff5e34a9fde17e9856d82cf5a8f3a473081008c7c2e64de2eeab4f81fde3c9",
        "domain": "numina",
        "question": "5.6. Among all the irreducible fractions, where the numerator and denominator are two-digit numbers, find the smallest fraction greater than $\\frac{4}{9}$. In your answer, specify its numerator.",
        "answer": "41",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 6465,
        "difficulty": null
      },
      {
        "id": "8db7d071b9a9676c19500e133434f4f8176e476c012109b9855850fd57d62665",
        "domain": "numina",
        "question": "When Bořek was painting the garage door, he accidentally painted over the scale of the outdoor wall thermometer. However, the mercury tube remained undamaged, so Bořek covered the original scale with a homemade strip. On it, he carefully marked divisions, all of the same size and labeled with numbers. His division, however, had a different size than the original division, which represented one degree Celsius, and he also placed the zero differently than where $0{ }^{\\circ} \\mathrm{C}$ was. Thus, Bořek began measuring temperature in his own units: borcs. When the thermometer should have indicated a temperature of $11^{\\circ} \\mathrm{C}$, it showed 2 borcs. When it should have indicated $-4{ }^{\\circ} \\mathrm{C}$, it showed -8 borcs. What is the temperature in degrees Celsius if Bořek sees a temperature of -2 borcs on his thermometer?\n\n(L. Simünek)",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76263,
        "difficulty": null
      },
      {
        "id": "c0448ac802003e89421ba5f303ed445f085fc0417169d86d95ce5d338c70a39b",
        "domain": "numina",
        "question": "The Law of Cosines\n[Pythagorean Theorem (direct and inverse).]\n\nIn a right triangle \\(ABC\\), the leg \\(AC = 15\\) and the leg \\(BC = 20\\). On the hypotenuse \\(AB\\), a segment \\(AD\\) equal to 4 is laid out, and point \\(D\\) is connected to \\(C\\). Find \\(CD\\).",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57952,
        "difficulty": null
      },
      {
        "id": "3fd391285ce23c53da987e5da8778e185209a2d6ce44ad73c476dcae70cf00be",
        "domain": "numina",
        "question": "6.24 In a geometric sequence with a common ratio greater than 1, what is the maximum number of terms that are integers between 100 and 1000.\n(4th Canadian Mathematics Competition, 1972)",
        "answer": "6",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 144214,
        "difficulty": null
      },
      {
        "id": "e0c973f367cf327e40b877df4531a1efc1be59438d2030d003585933cc2233da",
        "domain": "numina",
        "question": "Consider 100 points in the plane, such that no three are collinear. We draw all the lines passing through two of these points, and we assume that if $A, B, C, D$ are four of these points, then $(A B)$ and $(C D)$ are not parallel, and their intersection point does not lie on another line. How many regions of the plane are thus formed by these lines?",
        "answer": "11778426",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 82180,
        "difficulty": null
      },
      {
        "id": "71672f87ee89ba13b9c5ddc17f2cc11143f16af4792937f962f73d5417d3f400",
        "domain": "numina",
        "question": "543. a) How can a right triangle be divided into two similar triangles? b) In how many ways can this be done?",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 40285,
        "difficulty": null
      },
      {
        "id": "3ef28fce0e46bf2152a7b603c1667827934fd5e3ce6ff73e45ee514a10d54b8e",
        "domain": "numina",
        "question": "13. (2005 National High School Mathematics League Additional Question) Let positive numbers $a$, $b$, $c$, $x$, $y$, $z$ satisfy $c y + b z = a$; $a z + c x = b$; $b x + a y = c$. Find the minimum value of the function $f(x, y, z) = \\frac{x^2}{1 + x} + \\frac{y^2}{1 + y} + \\frac{z^2}{1 + z}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124266,
        "difficulty": null
      },
      {
        "id": "9788d5b68b7f81c5bccc51e7bd664003ef0fbacd2c74e178ad00d4295e2d1ed8",
        "domain": "numina",
        "question": "A dart is thrown at a square dartboard of side length 2 so that it hits completely randomly. What is the probability that it hits closer to the center than any corner, but within a distance 1 of a corner?",
        "answer": "\\frac{\\pi-2}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151743,
        "difficulty": null
      },
      {
        "id": "86445d9e44b10da56ba7f5e6006d639e595cb926c52c55b6bb2214df4a46ff7b",
        "domain": "numina",
        "question": "14. As shown in the figure, four rectangles, Jia, Yi, Bing, and Ding, are pieced together to form square $E F G H$, with the middle shaded area being a square. It is known that the sum of the areas of rectangles Jia, Yi, Bing, and Ding is $32 \\mathrm{~cm}^{2}$, and the area of quadrilateral $A B C D$ is $20 \\mathrm{~cm}^{2}$. Find the total perimeter of rectangles Jia, Yi, Bing, and Ding in centimeters.",
        "answer": "48",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133899,
        "difficulty": null
      },
      {
        "id": "518223054ece608d245e92d4c6416c603c720795aea78e69508e6b45d7bf12c4",
        "domain": "numina",
        "question": "7. If (1) $a, b, c, d$ all belong to $\\{1,2,3,4\\}$; (2) $a \\neq b, b \\neq c, c \\neq d, d \\neq a$; (3) $a$ is the smallest value among $a, b, c, d$, then the number of different four-digit numbers $\\overline{a b c d}$ that can be formed is $\\qquad$ .",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 147066,
        "difficulty": null
      },
      {
        "id": "2ea91d5933db5b3cb860016ccd6e2969d6d356ddc43d77b3b9c439ec99f9cb3f",
        "domain": "numina",
        "question": "14. Gauss and his classmates were punished at school for being mischievous, and the teacher assigned a math problem:\nFrom the natural numbers 1 to 100, remove two consecutive odd numbers, the average of the remaining numbers is 51, what is the product of the two removed odd numbers?\nWhoever answers correctly can go home. Gauss once again quickly gave the correct answer. Gauss's answer is $\\qquad$.",
        "answer": "675",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 137612,
        "difficulty": null
      },
      {
        "id": "e46ee35a607163721cb69500506e0b9891fb4649fe16f06fa698a8f3216ed181",
        "domain": "numina",
        "question": "45. Given a 129-gon. Petya and Vasya are playing a game. They take turns marking the vertices of this polygon, with Petya making the first move. Petya can mark any unmarked vertex on each of his turns. Vasya can mark any unmarked vertex that is two positions away from the vertex Petya marked on his last turn. The game ends when Vasya cannot make his move. What is the maximum number of moves Vasya can make regardless of Petya's play?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19036,
        "difficulty": null
      },
      {
        "id": "ef5e1d86f3f49e44608ff1ba03b111ca847f3d124165d94983801d644a0c6e13",
        "domain": "numina",
        "question": "6. A circle of radius $t$ is tangent to the hypotenuse, the incircle, and one leg of an isosceles right triangle with inradius $r=1+\\sin \\frac{\\pi}{8}$. Find $r t$.",
        "answer": "\\frac{2+\\sqrt{2}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152410,
        "difficulty": null
      },
      {
        "id": "257cdd25cd29c42845b7c424a6d6de1042dac47f76c436193b6544a9c4ac3005",
        "domain": "numina",
        "question": "6. Solve the equation $\\sqrt{\\frac{x^{2}-16}{x-3}}+\\sqrt{x+3}=\\frac{7}{\\sqrt{x-3}}$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19215,
        "difficulty": null
      },
      {
        "id": "4f99a166c98a1d26208ed579435d1ab05955c9eb85245a919e089c69718bfdc7",
        "domain": "numina",
        "question": "In the sequence 1, 3, 2, .. each term after the first two is equal to the preceding term, subtracted from the term that precedes it, that is, if $n>2$, then $a_{n}=a_{n-1}-a_{n-2}$. What is the sum of the first hundred terms of this sequence?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74495,
        "difficulty": null
      },
      {
        "id": "054cbb6f6f99eab2de7f1bbafa25f2e23a35a727fcaf7a0912b9411b6becf18a",
        "domain": "numina",
        "question": "Example 7 A positive integer $n$ cannot be divisible by 2 or 3, and there do not exist non-negative integers $a, b$ such that $\\left|2^{a}-3^{b}\\right|=n$. Find the minimum value of $n$.\n(2003 China Training Team Test)",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 145517,
        "difficulty": null
      },
      {
        "id": "a9ce5faf67de079ad44ebb84cbe983441880a179eadb5ade5a366a41c804a957",
        "domain": "numina",
        "question": "Question 4. Calculate\n$$\n\\log _{2}\\left(\\prod_{a=0}^{2018} \\prod_{b=0}^{2018}\\left(1+e^{\\frac{2 \\pi n k b}{2017}}\\right)\\right)\n$$",
        "answer": "6725",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101824,
        "difficulty": null
      },
      {
        "id": "c5b221db85ccd6404c2bb7c7403c3188e5160ad1544b8b6182be6aafa9e4714d",
        "domain": "numina",
        "question": "3. As shown in the figure, in trapezoid $A B C D$ with an area of 112, $A E: E F: F B=1: 3: 3, A D: B C=2$ : 3, then the area of triangle $C D H$ is $\\qquad$.",
        "answer": "36",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 145269,
        "difficulty": null
      },
      {
        "id": "a73a759b8774d266e4c0cabe9c0de8050959e3bc5a2927b62a753a6d6ff2fb96",
        "domain": "numina",
        "question": "5. In the country of Betia, there are 125 cities, some of which are connected by express trains that do not stop at intermediate stations. It is known that any four cities can be toured in a circle in some order. What is the minimum number of pairs of cities connected by express trains?",
        "answer": "7688",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 7951,
        "difficulty": null
      },
      {
        "id": "5ab4031bb015a3d269b801be462a855bf7bfe730846493ff290369187a46e100",
        "domain": "numina",
        "question": "B1. Arthur is driving to David's house intending to arrive at a certain time. If he drives at 60 $\\mathrm{km} / \\mathrm{h}$, he will arrive 5 minutes late. If he drives at $90 \\mathrm{~km} / \\mathrm{h}$, he will arrive 5 minutes early. If he drives at $n \\mathrm{~km} / \\mathrm{h}$, he will arrive exactly on time. What is the value of $n$ ?",
        "answer": "72",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120330,
        "difficulty": null
      },
      {
        "id": "13fe95e57b46ddde7b3126a4b99597bb6a9a559045cbbeca07fb85c349904061",
        "domain": "numina",
        "question": "1. Rectangle A has length $6 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nRectangle B has length $12 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nRectangle $\\mathrm{C}$ has length $9 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nThe rectangle with the smallest width has a width of $x \\mathrm{~cm}$. What is the value of $x$ ?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 146224,
        "difficulty": null
      },
      {
        "id": "f77bced17f2f6751c1559d8a68d884622c58accfe2daa8e9985755a6c6e09847",
        "domain": "numina",
        "question": "M . B.\n\nOn some cells of a $10 \\times 10$ board, there is a flea. Once a minute, the fleas jump simultaneously, and each one jumps to an adjacent cell (along a side). A flea jumps strictly in one of the four directions parallel to the sides of the board, maintains its direction as long as possible, and changes it to the opposite direction if it cannot continue. Dog Barboss observed the fleas for an hour and never saw two of them sitting on the same cell. What is the maximum number of fleas that could have been jumping on the board?",
        "answer": "40",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55753,
        "difficulty": null
      },
      {
        "id": "0eb5d9b877a1af15982b920be0b069818826bc90e74d1cbfdbee295eb1ceb53c",
        "domain": "numina",
        "question": "9.3. The number 2019 is represented as the sum of different odd natural numbers. What is the maximum possible number of addends?",
        "answer": "43",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17280,
        "difficulty": null
      },
      {
        "id": "204e3b5b6c851d4488246608b812e429887b12579bba0fc42fbd370811d78704",
        "domain": "numina",
        "question": "A pedestrian set out from point $A$ to point $B$. At the same time, a cyclist set out from $B$ to $A$. After one hour, the pedestrian found himself exactly halfway between point $A$ and the cyclist. Another 15 minutes later, they met, and each continued on their way.\n\nHow much time did the pedestrian spend on the journey from $A$ to $B$? (The speeds of the pedestrian and the cyclist are constant.)\n\n#",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 68955,
        "difficulty": null
      },
      {
        "id": "4b71a39bd9c3dcb6914dfd4df5b92ab60f43d58144a0bb4167a05ab3b0e6019a",
        "domain": "numina",
        "question": "## - Problem 1\n\nA pucelana sequence is a strictly increasing sequence of sixteen consecutive positive odd numbers, whose sum is a perfect cube. How many pucelana sequences consist only of three-digit numbers?",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 86371,
        "difficulty": null
      },
      {
        "id": "9920988278d49bc81aa9409cb02503b2ee3f75c13007efd43a7b47548e9c37c2",
        "domain": "numina",
        "question": "7.7. The distances from the midpoint of the height of a regular triangular pyramid to the lateral face and to the lateral edge are 2 and $\\sqrt{13}$, respectively. Find the volume of the pyramid. If necessary, round your answer to two decimal places.",
        "answer": "432.99",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6426,
        "difficulty": null
      },
      {
        "id": "758490a4aad0ca2d54306c25034e694e363d58f843c3162cf94cd89b83f99d91",
        "domain": "numina",
        "question": "On a table are 10 coins, all with \"heads\" facing up. A move consists of flipping exactly 4 coins.\n\na) What is the minimum number of moves required for all to be \"tails\" facing up?\n\nb) If there were 11 coins, would it be possible to have all of them with tails facing up?",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 73603,
        "difficulty": null
      },
      {
        "id": "d91fe23860dc1d0ddbeff9024607beb11aa297f2df3a9c28b79b8491e3916d2a",
        "domain": "numina",
        "question": "6. Given $A=\\{1,-1, \\mathrm{i},-\\mathrm{i}\\}$ (i is the imaginary unit), $f(x)$ is a function with domain and range both being $A$, for any $x, y \\in A$, there is $f(x y)=f(x) f(y)$. Then the number of functions $f(x)$ that satisfy the condition is $\\qquad$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97352,
        "difficulty": null
      },
      {
        "id": "49057e24e14406a94234f88bc35a65d27e5d33b19a5f88f6eafd2cac4897561c",
        "domain": "numina",
        "question": "7.1. Does there exist a ten-digit number, divisible by 11, in which all digits from 0 to 9 appear?",
        "answer": "9576843210",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15045,
        "difficulty": null
      },
      {
        "id": "5a41400e1724e039329a05620ee6b8c36366cb99018814e4d73cdcbd3eed105a",
        "domain": "numina",
        "question": "1. Given $a>0, b>0$, and $\\log _{9} a=\\log _{12} b=\\log _{16}(a+b)$. Then the value of $\\frac{b}{a}$ is $\\qquad$ .",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119785,
        "difficulty": null
      },
      {
        "id": "ccd00e7081ec61f2e0c16390437317629233ca4589649fb27801a80ba1c4739f",
        "domain": "numina",
        "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-5.5,5.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
        "answer": "132",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19652,
        "difficulty": null
      },
      {
        "id": "ba0b210ff4daebb02ab3bd48686c65d1c61655b91aad6727dbff45d1274f83dc",
        "domain": "numina",
        "question": "Find the height of a triangular pyramid, the lateral edges of which are pairwise perpendicular and equal to 2, 3, and 4.\n\n#",
        "answer": "\\frac{12}{\\sqrt{61}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63570,
        "difficulty": null
      },
      {
        "id": "c0417047cf08f35b7d34fbf92289f79d8ebc45e3fd10652f851ca8edf171bac9",
        "domain": "numina",
        "question": "10.366 Two circles touch each other externally at point $A$. Find the radii of the circles if the chords connecting point $A$ with the points of tangency of one of their common external tangents are 6 and 8 cm.",
        "answer": "\\frac{15}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 52055,
        "difficulty": null
      },
      {
        "id": "dce0bd56bd5efe27e3da7b647f39a27597bedff4d3cc6c8ed5fe33a3ad54e55e",
        "domain": "numina",
        "question": "9. The equation $9 x^{3}-3 x^{2}-3 x-1=0$ has a real root of the form $\\frac{\\sqrt[3]{a}+\\sqrt[3]{b}+1}{c}$, where $a, b, c$ are positive integers. Find the value of $a+b+c$.\n(1 mark)\n方程 $9 x^{3}-3 x^{2}-3 x-1=0$ 有一個形如 $\\frac{\\sqrt[3]{a}+\\sqrt[3]{b}+1}{c}$ （其中 $a \\cdot b \\cdot c$ 為正整數）的實根。求 $a+b+c$ 的值。\n(1 分)",
        "answer": "119",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153678,
        "difficulty": null
      },
      {
        "id": "3dc1ad2b0c9c4effd18e1a062b9d9e0897b6e50dd480ecad1247152137414ecb",
        "domain": "numina",
        "question": "4. In triangle $\\mathrm{ABC}$ with sides $a, b$ and $c$, it is known that $b=50 \\mathrm{~cm}, c=48 \\mathrm{~cm}$ and the internal angle at vertex A is twice the internal angle at vertex B. Determine the side $a$.",
        "answer": "70",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 189210,
        "difficulty": null
      },
      {
        "id": "a46fa133603e78dd068251d64c61bc1aead4ced1f3810e4d17d5907395fc8aad",
        "domain": "numina",
        "question": "9. In order to effectively improve the \"sandstorm\" phenomenon, a region attaches great importance to environmental protection, and the area of green vegetation is on the rise. According to the survey, from 1986 to 1995 over 10 years, it increased by $2 \\%$ every two years, and in 1994 and 1995, 8.15 million $\\mathrm{m}^{2}$ of vegetation was planted. The local government has decided to continue developing at this rate for the next 10 years. Therefore, from 1996 to 1999, the total area of green vegetation planted will be _____ million $\\mathrm{m}^{2}$ (accurate to the nearest million $\\mathrm{m}^{2}$).",
        "answer": "1679",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97844,
        "difficulty": null
      },
      {
        "id": "c5ab56d5bd5b80585d780db732c3127e48078ddc861d5f26cc2dcaa5d14c6bfe",
        "domain": "numina",
        "question": "1. Given is $\\triangle A B C$, over whose side $A C$ a circle $k$ is constructed as if over a diameter. The circle $k$ passes through the midpoint of side $B C$, and intersects side $A B$ at point $D$ such that $A D=\\frac{3}{2} D B$. If $A C=60$, calculate the area of $\\triangle A B C$.",
        "answer": "1440",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 161152,
        "difficulty": null
      },
      {
        "id": "1bee1ec4030d0dfbff3c599b03105d6b8fe9048a1355a229b1f868fd75a9c36a",
        "domain": "numina",
        "question": "Some people are playing paintball against each other. In a given situation, the distances between them are all different. In this case, everyone shoots at the person closest to them. What is the maximum number of people who can shoot at the same person?",
        "answer": "5",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 169653,
        "difficulty": null
      },
      {
        "id": "e24f8cf9588e8b63bd8d749b46485706404b9e662f36d2ad1907b270624b27ba",
        "domain": "numina",
        "question": "4. In the coordinate plane, there are 100 points. Let $N$ be the number of triples of points $(A, B, C)$ that satisfy the following conditions: the vertices are chosen from these 100 points, the y-coordinates of $A$ and $B$ are the same, and the x-coordinates of $B$ and $C$ are the same. Find the maximum value of $N$.",
        "answer": "8100",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 116998,
        "difficulty": null
      },
      {
        "id": "8c31c6e41ec777863996aa23d0092fe097803b28ed661b1e5c8744773b2ec5c3",
        "domain": "numina",
        "question": "12. Following the order of King Pea, General Mushtralkin tried to arrange all the soldiers in rows first by 2, and then by $3,4,5,6,7,8,9,10$, but to his surprise, each time the last row was incomplete, as there were respectively\n\n$$\n1,2,3,4,5,6,7,8,9\n$$\n\nsoldiers left. What is the smallest number of soldiers there could have been?",
        "answer": "2519",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39488,
        "difficulty": null
      },
      {
        "id": "6a839df5745ffaeaebe897a191275db0f838b482d1a71a8f2be25a02abb5ff93",
        "domain": "numina",
        "question": "B1. Find all real numbers $a \\geq 0$ for which the equation $2|x-a|+3|x+a|=1$ has at least one real solution.",
        "answer": "\\frac{1}{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163201,
        "difficulty": null
      },
      {
        "id": "7148881793bb84dda13c058636c0b301bceec8a3f5b49c02d1b0173e5c80dfad",
        "domain": "numina",
        "question": "191*. The sequence of natural numbers $a_{0}, a_{1}$, $a_{2}, a_{3}, \\ldots$ is formed according to the following rule:\n\n$$\na_{0}, a_{1}, a_{2}=\\left|a_{0}-a_{1}\\right|, a_{3}=\\left|a_{1}-a_{2}\\right|, \\ldots\n$$\n\n(and generally $a_{n}=\\left|a_{n-2}-a_{n-1}\\right|$ for all $n \\geqslant 2$); the sequence continues until the first zero. It is known that each number in the sequence does not exceed 1967. What is the maximum number of terms that such a sequence can contain?",
        "answer": "1967",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 29458,
        "difficulty": null
      },
      {
        "id": "6f7fd1686be6f277727f9a8d22daea5709b11a0fc7025554dec60d483842c6f6",
        "domain": "numina",
        "question": "11.005. The plane angle at the vertex of a regular triangular pyramid is $90^{\\circ}$. Find the ratio of the lateral surface area of the pyramid to the area of its base.",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51302,
        "difficulty": null
      },
      {
        "id": "5fccd7ebb129e4a58d5ab0a23e989678e36bf9cc265ea5eaa95ced0e48e9e6c2",
        "domain": "numina",
        "question": "12.022. Find the angle of a triangle if it is known that the sides enclosing this angle are 1 and 3, and the bisector of the angle is \\(0.75 \\sqrt{3}\\).",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 51407,
        "difficulty": null
      },
      {
        "id": "96258b4cce84a2336ef5aef6f8a437746a6ba14e2044cc91871710e82e207ab9",
        "domain": "numina",
        "question": "14. New Year's Problem (recommended for 8th grade, 4 points). On the New Year's table, there are 4 glasses in a row: the first and third are filled with orange juice, while the second and fourth are empty. While waiting for guests, Vanya absent-mindedly and randomly pours the juice from one glass to another. In one move, she can take a full glass and pour all its contents into one of the two empty glasses.\n\nFind the expected number of pourings after which, for the first time, everything will be the opposite: the first and third glasses will be empty, and the second and fourth will be full.",
        "answer": "6",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 10806,
        "difficulty": null
      },
      {
        "id": "537ffe43802e712c54bdbac914a3e3f0a13ff28b1853a7ac6ae187e635408f4e",
        "domain": "numina",
        "question": "10. Let $\\triangle A B C$ be equilateral, and let $D, E, F$ be points on sides $B C, C A, A B$ respectively, with $F A=9, A E=E C=6, C D=4$. Determine the measure (in degrees) of $\\angle D E F$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152150,
        "difficulty": null
      },
      {
        "id": "bbb50e0a7380aa8858bf25083844c3a5ad902df5d560b3f6662d0180db8b21a0",
        "domain": "numina",
        "question": "4. The three sides of a triangle can be expressed as $m^{2}+m+1, 2m+1, m^{2}-1$, then the largest angle of this triangle is . $\\qquad$",
        "answer": "\\frac{2\\pi}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97897,
        "difficulty": null
      },
      {
        "id": "ca73abf40b4e7de66b159dc7b3a4d0a0522691fea5d17442a23fe329cb100166",
        "domain": "numina",
        "question": "(11) Let $0<x<1,0<y<1$, and $(1-x y)^{2}=2(1-x)(1-y)$. Then the maximum value of the function $f(x, y)=\\frac{1}{2} x y(1-x y)$ is $\\qquad$ .",
        "answer": "5\\sqrt{2}-7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 142918,
        "difficulty": null
      },
      {
        "id": "dc55e6877c9ab7279b32ca46fda976b63ef038f69abb97de4e954459cc29190b",
        "domain": "numina",
        "question": "Exercise 6. Find the largest integer $n \\geqslant 3$ for which there exists a set $\\mathcal{S}$ of $n$ points in the plane with the following property: every triangle (even degenerate) whose vertices belong to $\\mathcal{S}$ is isosceles but not equilateral.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 77296,
        "difficulty": null
      },
      {
        "id": "c21a1bd4b85abef9f09a51cf5afcc5558d4a2efe6ddb9d2bcbb91c6e80de84f8",
        "domain": "numina",
        "question": "4. A box contains two white socks, three blue socks and four grey socks. Rachel knows that three of the socks have holes in, but does not know what colour these socks are. She takes one sock at a time from the box without looking. How many socks must she take for her to be certain she has a pair of socks of the same colour without holes?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 156412,
        "difficulty": null
      },
      {
        "id": "63443dabfe4a6b2aec201ea7c501de6c79e7d3adfeeccb75690dc8a9f10f4a1e",
        "domain": "numina",
        "question": "Example 5 Find the maximum constant $k$, such that $\\frac{k a b c}{a+b+c} \\leqslant(a+b)^{2}+(a+b+4 c)^{2}$ holds for all positive real numbers $a, b, c$.",
        "answer": "100",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 144513,
        "difficulty": null
      },
      {
        "id": "903aa837c7407a1487c8ce38994e4ff5e3cd988190fdb9c8b1e3880b9b17e0a9",
        "domain": "numina",
        "question": "Example 4. Solve the equation\n\n$$\n\\left|\\frac{x^{2}-6 \\sqrt{x}+7}{x^{2}+6 \\sqrt{x}+7}\\right|=1\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 40479,
        "difficulty": null
      },
      {
        "id": "a6a8acdb35a9b306169e7faf13d5ae1e908210ce7219954551a0211cd046524b",
        "domain": "numina",
        "question": "You are traveling in a foreign country whose currency consists of five different-looking kinds of coins. You have several of each coin in your pocket. You remember that the coins are worth 1, 2, 5, 10, and 20 florins, but you have no idea which coin is which and you don't speak the local language. You find a vending machine where a single candy can be bought for 1 florin: you insert any kind of coin, and receive\n\n1 candy plus any change owed. You can only buy one candy at a time, but you can buy as many as you want, one after the other.\n\nWhat is the least number of candies that you must buy to ensure that you can determine the values of all the coins? Prove that your answer is correct.\n\n#",
        "answer": "4",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 90919,
        "difficulty": null
      },
      {
        "id": "5b1b27b46b2fb04b0650700dfd080808063c17e5c7414d03668554ada1f917e9",
        "domain": "numina",
        "question": "Determine the largest real number $a$ such that for all $n \\geqslant 1$ and for all real numbers $x_{0}, x_{1}, \\ldots, x_{n}$ satisfying $0=x_{0}<x_{1}<x_{2}<\\cdots<x_{n}$, we have  \n$$ \\frac{1}{x_{1}-x_{0}}+\\frac{1}{x_{2}-x_{1}}+\\cdots+\\frac{1}{x_{n}-x_{n-1}} \\geqslant a\\left(\\frac{2}{x_{1}}+\\frac{3}{x_{2}}+\\cdots+\\frac{n+1}{x_{n}}\\right) . $$",
        "answer": "\\frac{4}{9}",
        "topic": "Inequalities",
        "source": "olympiads_ref",
        "source_index": 196272,
        "difficulty": null
      },
      {
        "id": "98d5061ad7592a63ed6d1fd6c1a896d8e22300f57a593503f9c0757374d15b74",
        "domain": "numina",
        "question": "7.5. Determine the integer numbers $x$, for which the numbers $n=x^{2}+5 x+1$ and $m=x^{2}+3 x+$ 7 are simultaneously perfect squares.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85113,
        "difficulty": null
      },
      {
        "id": "096e3d47943d923d6cc549d37a23e15a50c54dd5e60af31fa4daa09095882b2d",
        "domain": "numina",
        "question": "Find all strictly positive numbers $a, p, n$ such that $p^{a}-1=$ $2^{n}(p-1)$, where $p$ is a prime number.",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 82823,
        "difficulty": null
      },
      {
        "id": "b8588dd45554ba7a7f1c8f4c8172e780a7dfd435ee38dbccf360f9cbf8684e92",
        "domain": "numina",
        "question": "7. Suppose $A B C D$ is an isosceles trapezoid in which $\\overline{A B} \\| \\overline{C D}$. Two mutually externally tangent circles $\\omega_{1}$ and $\\omega_{2}$ are inscribed in $A B C D$ such that $\\omega_{1}$ is tangent to $\\overline{A B}, \\overline{B C}$, and $\\overline{C D}$ while $\\omega_{2}$ is tangent to $\\overline{A B}, \\overline{D A}$, and $\\overline{C D}$. Given that $A B=1, C D=6$, compute the radius of either circle.",
        "answer": "\\frac{3}{7}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152411,
        "difficulty": null
      },
      {
        "id": "fce909bdc098044107c132acf6954ef34535338ff90ab7eddf6947ad91090a89",
        "domain": "numina",
        "question": "If the price of a product increased from 5.00 to 5.55 reais, what was the percentage increase?",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74414,
        "difficulty": null
      },
      {
        "id": "add5ac210864251c014cdebae84ac3ea9561249f568398a8768c7eb4d7474e26",
        "domain": "numina",
        "question": "11. (15 points) In the sequence $\\left\\{a_{n}\\right\\}$, $a_{1}, a_{2}$ are given non-zero integers, $a_{n+2}=\\left|a_{n+1}-a_{n}\\right|$.\n（1）If $a_{15}=2, a_{16}=-1$, find $a_{2008}$;\n（2）Prove: From $\\left\\{a_{n}\\right\\}$, it is always possible to select infinitely many terms to form two different constant sequences.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 138506,
        "difficulty": null
      },
      {
        "id": "52de87d70cf7fe818a72f1c9498a662b573fa0ce39d9ec240d3a82727824b15d",
        "domain": "numina",
        "question": "## Task 7\n\nThe students of grades 1, 2, 3, and 4 of the Karl-Marx-Oberschule have committed to donating 300 marks for solidarity this school year.\n\nGrade 1 donated $48.70 \\mathrm{M}$, Grade 2 donated $40.50 \\mathrm{M}$, Grade 3 donated $53.20 \\mathrm{M}$, and Grade 4 donated $61.80 \\mathrm{M}$.\n\na) How many marks have all 4 classes donated in total?\n\nb) How many marks still need to be donated to fulfill the commitment?",
        "answer": "204.20",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 186676,
        "difficulty": null
      },
      {
        "id": "c99e6c234d902b4f4fbf4e588c27679a322e05d806766b9c1d72b463259f2da4",
        "domain": "numina",
        "question": "# 4. Variant 1\n\nThe train was moving from point A to point B at a constant speed. Halfway through the journey, a breakdown occurred, and the train stopped for 15 minutes. After that, the driver had to increase the train's speed by 4 times to arrive at point B on schedule. How many minutes does the train travel from point A to point B according to the schedule?",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16273,
        "difficulty": null
      },
      {
        "id": "c8fefe8490652512869eed058868a6d4cad820026c28ba4a17ff21a75f0bec79",
        "domain": "numina",
        "question": "21. Let's take any 1962-digit number divisible by 9. The sum of its digits will be denoted by $a$, the sum of the digits of $a$ by $b$, and the sum of the digits of $b$ by $c$. What is $c$?[^0]",
        "answer": "9",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 25230,
        "difficulty": null
      },
      {
        "id": "bb302c1e842c17a2a878eb4e09da1f7fa9120f33a99d5c15e5d3cb1c9dc1c85f",
        "domain": "numina",
        "question": "2. The numbers 1059, 1417, and 2312 leave the same remainder when divided by $d$, and $d>1$, find $d$.",
        "answer": "179",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112754,
        "difficulty": null
      },
      {
        "id": "6768a83677ef3bad783a82f656e83fb2cf877be73a39bd82f6596826700a93b3",
        "domain": "numina",
        "question": "5. Arithmetic density. It is clear that if we take a random natural number from a sufficiently long segment of the natural number series, the probability that this number will be divisible by 10 will be closer to 0.1, the longer the segment.\n\nLet's ask a different question: how much can this probability differ from 0.1? It can be 0, for example, if we specified the segment $1,2,3$, where no number is divisible by 10. And what is:\n\na) (recommended for 6th grade, 1 point) the greatest possible probability;\n\nb) (recommended for 6th grade, 1 point) the smallest possible non-zero probability?",
        "answer": "\\frac{1}{19}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 10797,
        "difficulty": null
      },
      {
        "id": "b93b3ccd79af852ad34c1bd158eb66c786201556521264aa462f1b7ff4d0c5c0",
        "domain": "numina",
        "question": "In the parking lot, there were cars and bicycles. If one more car arrived, there would be as many of them as there are bicycles. If five more bicycles arrived, all the bicycles would have as many wheels as all the cars.\n\nHow many cars and how many bicycles were in the parking lot?\n\n(M. Dillingerová)\n\nHint. Imagine a situation where the number of wheels of the cars and bicycles is the same.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76583,
        "difficulty": null
      },
      {
        "id": "bbde4a681c1fd81a547a909e55863633b69f04a130f74a611babeab4372adfd6",
        "domain": "numina",
        "question": "Problem 2. Determine the integers $x, y$, for which\n\n$$\n5^{x}-\\log _{2}(y+3)=3^{y} \\text { and } 5^{y}-\\log _{2}(x+3)=3^{x}\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179180,
        "difficulty": null
      },
      {
        "id": "bcf63e58c1d06cf7fb119b5ff1b9aa878d231a3af55d1c40347ecf17af0c3d6a",
        "domain": "numina",
        "question": "6. Each boy is friends with 5 girls, while all girls are friends with a different number of boys. What is the minimum number of children that can be in this group?",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 21730,
        "difficulty": null
      },
      {
        "id": "6ff6a00fba497cac1b8a2ead659a9a836d690e830d7818c0f99436d1e112861e",
        "domain": "numina",
        "question": "For what value of $k$ is $k \\%$ of 25 equal to $20 \\%$ of 30 ?",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194443,
        "difficulty": null
      },
      {
        "id": "73adf3220c2913f266bbf0f23361c156e35296e61a5eed6504882cc56303bf0b",
        "domain": "numina",
        "question": "Problem 5. Three square paths with a common center are 1 m apart from each other (see figure). Three ants start simultaneously from the lower left corners of the paths and run at the same speed: Mu and Ra counterclockwise, and Wei clockwise. When Mu reaches the lower right corner of the largest path, the other two, who have not yet completed a full circle, are on the right sides of their paths, and all three are on the same straight line. Find the sides of the squares.\n\n[6 points] (A. V. Shapovalov)",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22880,
        "difficulty": null
      },
      {
        "id": "9869cd8b66efcd6f5b96b312d77a7e7efeed7dcc93e792877d4402edc6cfac40",
        "domain": "numina",
        "question": "It is known that the remainder of dividing a certain prime number by 60 is a composite number. Which one?",
        "answer": "49",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69660,
        "difficulty": null
      },
      {
        "id": "5f0f301f850a05f6270495ac67336ceb4063435b51f082d861eb80cfdb0f6367",
        "domain": "numina",
        "question": "25. For an exam answer, a student can receive one of the following grades: $5,4,3,2$. The probability that a student will receive a grade of 5 is 0.3; a grade of $4-0.4$; a grade of $3-0.2$ and a grade of $2-0.1$. Which of the named events form a complete group of mutually exclusive events? What event is opposite to the event: “the student will receive a grade of 5” and what is the probability of this event?",
        "answer": "0.7",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 31667,
        "difficulty": null
      },
      {
        "id": "b8397aba579efa075484dcabd07cefdfd9f57f313ea7e8e6783ce5c4ff3bdb05",
        "domain": "numina",
        "question": "Suppose we have a convex polygon in which all interior angles are integers when measured in degrees, and the interior angles at every two consecutive vertices differ by exactly $1^{\\circ}$. If the greatest and least interior angles in the polygon are $M^{\\circ}$ and $m^{\\circ}$, what is the maximum possible value of $M-m$ ?",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 90395,
        "difficulty": null
      },
      {
        "id": "78022919bda34b8f658e0eb9730f19deb3c955b2b7109f736eba3f330f7a98a6",
        "domain": "numina",
        "question": "Folklore\n\nOn a chessboard, $n$ white and $n$ black rooks are placed such that rooks of different colors do not attack each other. Find the maximum possible value of $n$.\n\n#",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 68964,
        "difficulty": null
      },
      {
        "id": "40191f13b954c98e958a8ebc4dd4ab0ec45d2521a08c62db18eb23dd4766b407",
        "domain": "numina",
        "question": "\\section*{Exercise 1 - 081221}\n\nGive all prime numbers \\(p\\) for which both \\(p+10\\) and \\(p+14\\) are also prime numbers!",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 185745,
        "difficulty": null
      },
      {
        "id": "ffe54a5162b70138957ff1742232299c4ca7ca83b9558709105d8f8b9c60a22f",
        "domain": "numina",
        "question": "1-110 Given the line $a x+b y+c=0$ where the coefficients $a, b, c$ are 3 different elements chosen from the set $\\{-3,-2,-1,0,1,2,3\\}$, and the inclination angle of the line is acute. Find the number of all such different lines.",
        "answer": "43",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 108164,
        "difficulty": null
      },
      {
        "id": "73cf60219159f9087ec342b0bea67ac789546ea0d64c91ec72aa5634984f776c",
        "domain": "numina",
        "question": "8. (10 points) As shown in the figure, $A B C D$ is a parallelogram. $A E=2 E B$. If the area of triangle $C E F$ is $=1$. Then, the area of parallelogram $A B C D$ is $\\qquad$ .",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 132355,
        "difficulty": null
      },
      {
        "id": "1cc05d57483b4ce588b3a78c0ccc9ab22cb243a63e138608d520d6dd5a23fd9f",
        "domain": "numina",
        "question": "1572. Let the germination rate of rye seeds be $90 \\%$. What is the probability that out of 7 sown seeds, 5 will germinate?",
        "answer": "0.124",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 30648,
        "difficulty": null
      },
      {
        "id": "e711342f613d070616238c466dbb4211306497d67ea0697dab54cbd347e77e1b",
        "domain": "numina",
        "question": "G2.2 In Figure 2(a), $A B C D$ is a rectangle. $D E: E C=1: 5$, and $D E=12^{\\frac{1}{4}} . \\triangle B C E$ is folded along the side BE. If $b$ is the area of the shaded part as shown in Figure 2(b), find the value of $b$.\n\nG2.3 Let the curve $y=x^{2}-7 x+12$ intersect the $x$-axis at points $A$ and $B$, and intersect the $y$-axis at $C$. If $c$ is the area of $\\triangle A B C$, find the value of $c$.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 152914,
        "difficulty": null
      },
      {
        "id": "40b9118ad16c55acc42194808ad2701bbf86c9b941bc4bfc1a459917e9c3f1e6",
        "domain": "numina",
        "question": "The cost of withdrawing money from an ATM consists of two parts. There is a base fee, which is independent of the amount withdrawn. In addition, there is a part that is directly proportional to the amount withdrawn. What is the cost of withdrawing $85000 \\mathrm{Ft}$, if the cost of withdrawing $40000 \\mathrm{Ft}$ is $221 \\mathrm{Ft}$, and the cost of withdrawing $100000 \\mathrm{Ft}$ is $485 \\mathrm{Ft}$?",
        "answer": "419",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 169496,
        "difficulty": null
      },
      {
        "id": "7f7e90b8a70052fd1cdf2489b76023025b723302c592720b5fcdf858dc0d3b09",
        "domain": "numina",
        "question": "Determine the value of $(666666666)^{2}-(333333333)^{2}$.",
        "answer": "333333332666666667",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74640,
        "difficulty": null
      },
      {
        "id": "5437920ba55b510bc717bf4e149be16ea0be18e34ce727a5944a3796c5f8ca13",
        "domain": "numina",
        "question": "Let's determine the smallest positive number divisible by 999 that does not contain the digit 9.",
        "answer": "111888",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 167797,
        "difficulty": null
      },
      {
        "id": "3e837b20ae76146eeb4dbf78defa7a282312aed8755f31704ce336125e1c3235",
        "domain": "numina",
        "question": "4. Let $A$ be any set of 11 real numbers, and let the set $B=\\{u v \\mid u, v \\in A, u \\neq v\\}$. Find the minimum number of elements in $B$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "17",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 113416,
        "difficulty": null
      },
      {
        "id": "8c1367ef96fcdf9944f419e5c384766aa13ce623cda97f65dd7f6f13f96cdc17",
        "domain": "numina",
        "question": "5. If the function $f(x)=\\frac{a+\\sin x}{2+\\cos x}+b \\tan x$ has a sum of its maximum and minimum values equal to 4, then $a+b=$ $\\qquad$ .",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103869,
        "difficulty": null
      },
      {
        "id": "a45370d16b7418b04060aaf57a49506ab8f6eeb1a20452b6dd807b7424130d5b",
        "domain": "numina",
        "question": "11. A food factory has made 4 different exquisite cards, and each bag of food produced by the factory randomly contains one card. It is stipulated that if one collects all 4 different cards, they can win a prize. Xiao Ming buys 6 bags of this food at once. What is the probability that Xiao Ming will win the prize? $\\qquad$ .",
        "answer": "\\frac{195}{512}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 122936,
        "difficulty": null
      },
      {
        "id": "0ebcf8e569555e8db72f4112776308d78e5e014392576fee92c558c77e8c9a70",
        "domain": "numina",
        "question": "Problem 4.4. Along the road connecting Masha's and Sasha's houses, there are 17 apple trees and 20 poplars. When Masha was going to visit Sasha, she took photos of all the trees. Right after the tenth apple tree, Masha's phone memory ran out, and she couldn't photograph the remaining 13 trees. The next day, when Sasha was going to visit Masha, starting from the eighth apple tree, he picked one leaf from each tree. How many leaves did Sasha pick?",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13232,
        "difficulty": null
      },
      {
        "id": "7dbe0fed1f22fe0a27f1404bff9a4d120784409dd33566a7684df299819edf4e",
        "domain": "numina",
        "question": "Gombóc Artúr lives at 1 Édes Street, while the chocolate shop is at the other end of the street, at number $n$. Every day, Artúr follows this fitness routine: he starts in front of the house numbered 2. If he stands in front of the house numbered $k$ (where $1<k<n$), he flips an expired but fair chocolate coin. If it lands heads, he moves to the house numbered ( $k-1$ ), and if it lands tails, he moves to the house numbered ( $k+1$ ). If he reaches the chocolate shop, he goes in and drinks a chocolate milkshake, then moves to the house numbered ( $n-1$ ). When he gets home, his workout is over. On average, how many chocolate milkshakes does Artúr drink each day?",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 169903,
        "difficulty": null
      },
      {
        "id": "0dc01faa2c3f8af96bc8027af24cda4de6282b247d5c2473566bc7a3fe496106",
        "domain": "numina",
        "question": "Problem 8.7. The figure shows two equal triangles: $A B C$ and $E B D$. It turns out that $\\angle D A E = \\angle D E A = 37^{\\circ}$. Find the angle $B A C$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-27.jpg?height=432&width=711&top_left_y=91&top_left_x=369)",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15966,
        "difficulty": null
      },
      {
        "id": "b1f351b6c613a61adf76997b9f47df46b60c30588e05cc884ecbcff8468ce831",
        "domain": "numina",
        "question": "3. If $n$ squares in a plane satisfy:\n(1) All squares are congruent;\n(2) If two squares have a common intersection point $P$, then $P$ is a vertex of each of the squares;\n(3) Each square intersects exactly three other squares, then this set of $n$ squares is called \"triconnected.\"\n\nQuestion: How many positive integers $n$ in $2018 \\leqslant n \\leqslant 3018$ allow for the existence of a triconnected set with $n$ squares?",
        "answer": "501",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 110636,
        "difficulty": null
      },
      {
        "id": "01690f110e8e6658a8c21ab83caf5bf34a2d94cd5cb91e0d4d27973493d41b20",
        "domain": "numina",
        "question": "Example 1: $n$ teams are to hold a home-and-away double round-robin match (each pair of teams plays two matches, each having one home match), and each team can play multiple away matches within a week (from Sunday to Saturday). However, if a team has a home match in a certain week, no away matches for that team can be scheduled within the same week. If all matches can be completed within 4 weeks, find the maximum value of $n$.\n(Note: A match held at Team A's venue is called Team A's home match and Team B's away match.)",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 138815,
        "difficulty": null
      },
      {
        "id": "8f51542723bb8087a7f54e92b01004ffacb9a3324f5d8d7daf13b8628a6e342d",
        "domain": "numina",
        "question": "Below is a part of a square grid consisting of 4 rows and 2023 columns.\n\nDetermine the number of squares, all of whose vertices are grid points of the square grid.\n\n(K. Pazourek)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_096aded327f4b802fe92g-7.jpg?height=337&width=1422&top_left_y=1214&top_left_x=317)",
        "answer": "40430",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 76697,
        "difficulty": null
      },
      {
        "id": "ebeec6bb14c11eaa023ec497e64c6667f148bbf97e643e448e7c756c5bd80171",
        "domain": "numina",
        "question": "In Jen's baseball league, each team plays exactly 6 games against each of the other teams in the league. If a total of 396 games are played, how many teams are in the league?\n(A) 12\n(B) 16\n(C) 15\n(D) 13\n(E) 9",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 192902,
        "difficulty": null
      },
      {
        "id": "bfeb99aba5706eded524475517af9131921d042c3d85e420ede040e6322f3ccc",
        "domain": "numina",
        "question": "24. What is the unit digit of $1^{3}+2^{3}+3^{3}+\\ldots+2017^{3}+2018^{3}$? (Note: $a^{3}=a \\times a \\times a$)",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 102362,
        "difficulty": null
      },
      {
        "id": "d3193abfe148984ee402ebd754350214257687b4774a57f5bc93ef0872a8e0eb",
        "domain": "numina",
        "question": "12. In the figure, there are two squares of different sizes. The area of the shaded part is 1.25 times the area of the blank part. When the side length of the larger square is $a$, the side length of the smaller square is",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 123546,
        "difficulty": null
      },
      {
        "id": "1d8a7d801450f072b25360bf1b2ed485c1525c9363a52d0d08ffab63708fb1c0",
        "domain": "numina",
        "question": "30. There are 700 cards of 6 different colors: red, orange, yellow, green, blue, and white in a box. The ratio of the number of red, orange, and yellow cards is $1: 3: 4$, and the ratio of the number of green, blue, and white cards is 3:1:6. It is known that there are 50 more yellow cards than blue cards. How many cards must be drawn to ensure that at least 60 cards of the same color are drawn?",
        "answer": "312",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 131966,
        "difficulty": null
      },
      {
        "id": "cfab64cb17fba17703853c17b980ec8900d25bcb481536188127a50314149387",
        "domain": "numina",
        "question": "Example 7 (2003 National High School Competition Question) Given that $a, b, c, d$ are all positive integers, and $\\log _{a} b=\\frac{3}{2}, \\log _{c} d=\\frac{5}{4}$, if $a-c=$ 9, then $b-d=$ $\\qquad$ .",
        "answer": "93",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92156,
        "difficulty": null
      },
      {
        "id": "a3295b7a1d27060800e900af8b40a1428324c4b90f3f7f10b7107badf0347db4",
        "domain": "numina",
        "question": "S1.4 Let $f(x)$ be a function such that $f(n)=(n-1) f(n-1)$ and $f(n) \\neq 0$ hold for all integers $n \\geq 6$.\nIf $U=\\frac{f(T)}{(T-1) f(T-3)}$, find the value of $U$.",
        "answer": "72",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153260,
        "difficulty": null
      },
      {
        "id": "05c5515952c9fe90ca13f7faca5fb7665063264caab71ada920ea1c755a53ab9",
        "domain": "numina",
        "question": "2. Positive integers $x, y$, for which $\\gcd(x, y)=3$, are the coordinates of a vertex of a square with its center at the origin and an area of $20 \\cdot \\text{lcm}(x, y)$. Find the perimeter of the square.",
        "answer": "24\\sqrt{5}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 2050,
        "difficulty": null
      },
      {
        "id": "3e855ee31489aa163f84e23f2015e0e5d92147276048bc31ba6ae35b3412cb06",
        "domain": "numina",
        "question": "Problem 10.2. Points $A, B, C, D, E, F, G$ are located clockwise on a circle, as shown in the figure. It is known that $A E$ is the diameter of the circle. Also, it is known that $\\angle A B F=81^{\\circ}, \\angle E D G=76^{\\circ}$. How many degrees does the angle $F C G$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-36.jpg?height=561&width=616&top_left_y=89&top_left_x=419)",
        "answer": "67",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15970,
        "difficulty": null
      },
      {
        "id": "c4c6ec79f33761b96d1bec21500fda9091e30f5ab79f57fa832ee04b459ac17f",
        "domain": "numina",
        "question": "Example 9 (2003 Bulgarian National Mathematical Olympiad) Find the number of real solutions of the system of equations $\\left\\{\\begin{array}{l}x+y+z=3 x y, \\\\ x^{2}+y^{2}+z^{2}=3 x z, \\\\ x^{3}+y^{3}+z^{3}=3 y z\\end{array}\\right.$.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 124547,
        "difficulty": null
      },
      {
        "id": "8db38a59d46f23e89d475f77320dfce20e13b8682f21e52d3bc6343a74ee8c23",
        "domain": "numina",
        "question": "Is there a regular polygon in which the length of the shortest diagonal is equal to the radius of the circumscribed circle of the polygon?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 170222,
        "difficulty": null
      },
      {
        "id": "ff672b718ae1b49b5e8144c63a2f9467589b41294d06932de811587f24ff5a88",
        "domain": "numina",
        "question": "## Task $6 / 83$\n\nIn every rectangle, the angle bisectors intersect at four points that span a square (if the rectangle is a square, these four points coincide).\n\nThe area $A_{Q}$ of this square is to be represented as a function of the side ratio $x=a: b$ (where $a>b, b$ is constant). For which side ratio is the square area $A_{Q}$ equal to the rectangle area $A_{R}$?",
        "answer": "2+\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 182701,
        "difficulty": null
      },
      {
        "id": "3cb8a19c8b955d5bbd10a3f473d101f3d5e7ac6edfb5a69c8ee4348f7cbe0ba6",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nFour line segments intersect in points $A, B, C, D$, and $E$, as shown. The measure of $\\angle C E D$ is $x^{\\circ}$. What is the value of $x$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_9c5b2a5bb2c0ebd3406fg-4.jpg?height=439&width=829&top_left_y=1447&top_left_x=651)\n\n##",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194122,
        "difficulty": null
      },
      {
        "id": "ea3625320275fc178b6b904243236abb5771fc7dc64304f3f0676d9e6cacbc1b",
        "domain": "numina",
        "question": "7. Given vectors $\\boldsymbol{a}, \\boldsymbol{b}, \\boldsymbol{c}$ satisfy\n$$\n\\begin{array}{l}\n|a|=|b|=3,|c|=4, \\boldsymbol{a} \\cdot \\boldsymbol{b}=-\\frac{7}{2}, \\\\\n\\boldsymbol{a} \\perp \\boldsymbol{c}, \\boldsymbol{b} \\perp \\boldsymbol{c} .\n\\end{array}\n$$\n\nThen for real numbers $x, y$, the expression\n$$\n|x \\boldsymbol{a}+y \\boldsymbol{b}+(1-x-y) \\boldsymbol{c}|\n$$\n\nhas the minimum value of $\\qquad$",
        "answer": "\\frac{4\\sqrt{33}}{15}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99675,
        "difficulty": null
      },
      {
        "id": "feb65f5cbcde4e3c6040d14433a58425481d02a3c80c5e7d7d066414f96a36d9",
        "domain": "numina",
        "question": "4. Find the area of the figure, the coordinates $(x ; y)$ of the points of which satisfy the system of inequalities\n\n$$\n\\left\\{\\begin{array}{l}\n|x|+|y| \\geq 1 \\\\\n(|x|-1)^{2}+(|y|-1)^{2} \\leq 1\n\\end{array}\\right.\n$$",
        "answer": "\\pi-2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11409,
        "difficulty": null
      },
      {
        "id": "0390cb06d2531214e58541e3a966ca040e61242dccab8ba5a43c5daf90ea0d3a",
        "domain": "numina",
        "question": "7. In the addition problem on the right, the sum of the digits of the addend is three times the sum of the digits of the sum. What is the smallest possible addend?\n\n\n\nThe above text has been translated into English, preserving the original text's line breaks and format.",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 143444,
        "difficulty": null
      },
      {
        "id": "25250d9aa123761fd358a8bd610de0d0dbfa3cf32d5843534262ee3879210661",
        "domain": "numina",
        "question": "3. If real numbers $x, y$ satisfy $4 x^{2}-4 x y+2 y^{2}=1$, then the sum of the maximum and minimum values of $3 x^{2}+x y+y^{2}$ is $\\qquad$",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109397,
        "difficulty": null
      },
      {
        "id": "228ed2c738ef427811913fc2310878230a35019e40a41fa6c583ce0e231f6c6a",
        "domain": "numina",
        "question": "$\\underline{\\text { Frankin B.R. }}$\n\nGiven a polynomial $P(x)$ with real coefficients. An infinite sequence of distinct natural numbers $a_{1}, a_{2}, a_{3}, \\ldots$ is such that\n\n$P\\left(a_{1}\\right)=0, P\\left(a_{2}\\right)=a_{1}, P\\left(a_{3}\\right)=a_{2}$, and so on. What degree can $P(x)$ have?",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 55432,
        "difficulty": null
      },
      {
        "id": "172152beea3144bd7a83e531547c5506ab5c62399117237236903137af7848ce",
        "domain": "numina",
        "question": "Example 24 (IMO-18 Problem) Find the maximum number of elements in a set $S$ that satisfies the following conditions:\n(1) Each element of $S$ is a positive integer not exceeding 100;\n(2) For any two distinct elements $a, b$ in $S$, there exists an element $c$ in $S$ such that the greatest common divisor of $a$ and $c$ is 1, and the greatest common divisor of $b$ and $c$ is also 1;\n(3) For any two distinct elements $a, b$ in $S$, there exists an element $d$ in $S$ different from $a, b$ such that the greatest common divisor of $a$ and $d$ is greater than 1, and the greatest common divisor of $b$ and $d$ is also greater than 1.",
        "answer": "72",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 117361,
        "difficulty": null
      },
      {
        "id": "69d2cc9de87808930c6cb2ef8b87f96e73caa3da2eb4b7e01b5686adae34b610",
        "domain": "numina",
        "question": "Bogdanov I.i.\n\nInitially, there are 111 pieces of plasticine of the same mass on the table. In one operation, you can choose several groups (possibly one) with the same number of pieces and in each group, combine all the plasticine into one piece. What is the minimum number of operations required to get exactly 11 pieces, each of which has a different mass?\n\n#",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 52747,
        "difficulty": null
      },
      {
        "id": "14e889d781fa1a58c8a3c05029f0c18bf8654c8dde47eed545b57893b7df9ca9",
        "domain": "numina",
        "question": "$$\n2015 \\times 2016\n$$\n1. (8 points) The calculation result of the expression $\\frac{1}{1+\\frac{1}{2016}}+\\frac{2}{2+\\frac{2}{2016}}+\\frac{3}{3+\\frac{3}{2016}}+\\cdots+\\frac{2015}{2015+\\frac{2015}{2016}}$ is",
        "answer": "2017",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133731,
        "difficulty": null
      },
      {
        "id": "7ced0d53b6f9cdf3d69eafa5729074a7003197f5cd39b16de0311375913944b7",
        "domain": "numina",
        "question": "There are 2017 unoccupied chairs in a row. Every minute, a person arrives and sits in one of them that is empty, and at the same instant, if it is occupied, a person in an adjacent chair stands up and leaves. What is the maximum number of people that can be sitting simultaneously in the row of chairs?",
        "answer": "2016",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 74592,
        "difficulty": null
      },
      {
        "id": "cffbbed8f6a3c145affd0f7add54e1642eefcfb85ee147be12c35df2cab7c713",
        "domain": "numina",
        "question": "Let $A, B, C$ be points in a line on this order, with $A B=B C$. A car travels from $A$ to $B$ at 40 kilometers per hour and then from $B$ to $C$ at 60 kilometers per hour, without stopping in between. What is the overall speed of the trip from $A$ to $C$ ?",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90653,
        "difficulty": null
      },
      {
        "id": "5dc666403550bd785e3751d04ad3d7dc8342d90d13a63050100c75c3f9ce5262",
        "domain": "numina",
        "question": "Exercise 7. Consider a $10 \\times 10$ grid, composed of 100 cells. The rows of the grid are numbered from top to bottom in the order of 1 to 10: the topmost row has the number 1, the bottommost row has the number 10. The columns of the grid are numbered from left to right in the order of 1 to 10: the leftmost column has the number 1, the rightmost column has the number 10. Martin colors some cells of the grid such that for any colored cell $c$, there is at most one different colored cell whose column number is greater than or equal to that of the column of $c$ and whose row number is greater than or equal to that of the row of $c$.\n\nWhat is the maximum number of cells that Martin can color?",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 78246,
        "difficulty": null
      },
      {
        "id": "f36afad454bb7fa8e43047947ed9c482dbe97edb85339b52fdcb71376d040c73",
        "domain": "numina",
        "question": "10. (This question is worth 20 points) The sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=\\frac{\\pi}{6}, a_{n+1}=\\arctan \\left(\\sec a_{n}\\right)\\left(n \\in \\mathbf{N}^{*}\\right)$. Find the positive integer $m$, such that\n$$\n\\sin a_{1} \\cdot \\sin a_{2} \\cdots \\cdot \\sin a_{m}=\\frac{1}{100} .\n$$",
        "answer": "3333",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94886,
        "difficulty": null
      },
      {
        "id": "841ff0888c560ec16283bd1f4f545234a9d3b1c07243fc4f1b24f007262b8409",
        "domain": "numina",
        "question": "15. Given that the function $f(x)$ is a decreasing function defined on $(-\\infty, 1]$, and for all real numbers $x$, the inequality $f(k-\\sin x) \\geqslant$ $f\\left(k^{2}-\\sin ^{2} x\\right)$ always holds, then the value of $k$ is $\\qquad$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146446,
        "difficulty": null
      },
      {
        "id": "b29e17974e0e2d0dfa7e2730b7cca6e6bfb630ea9185143379a7b3d55e60a2cd",
        "domain": "numina",
        "question": "G3.2 Suppose the equation $x^{2} y-x^{2}-3 y-14=0$ has only one positive integral solution $\\left(x_{0}, y_{0}\\right)$. If $x_{0}+y_{0}=b$, find the value of $b$.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152949,
        "difficulty": null
      },
      {
        "id": "c6cc7d8326dadf12163eea3979ca226699924707a2ca7b77f0cedffe2d4fe07f",
        "domain": "numina",
        "question": "7.4. In triangle $A B C$, the angles $A$ and $C$ at the base are $20^{\\circ}$ and $40^{\\circ}$, respectively. It is known that $A C - A B = 5$ (cm). Find the length of the angle bisector of angle $B$.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22478,
        "difficulty": null
      },
      {
        "id": "f7a66015d23887f90950ee1df3eebcc03c321c2f25ae8cc4bd897d313a711a6b",
        "domain": "numina",
        "question": "1. [3] Find the sum of all real solutions to $x^{2}+\\cos x=2019$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151784,
        "difficulty": null
      },
      {
        "id": "2828b35a38986f332687346b656fe377076404d3e5055f05a9e2bb618f5a91da",
        "domain": "numina",
        "question": "8. The maximum volume of a cone inscribed in a sphere compared to the volume of the sphere is $\\qquad$\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "\\frac{8}{27}",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 131281,
        "difficulty": null
      },
      {
        "id": "bb72d36dc942a0a31d753f9ff316fac31c813f752b555ed495f4cee930fa25ad",
        "domain": "numina",
        "question": "1. In a convex quadrilateral $ABCD$, $\\angle B=\\angle D=90^{\\circ}, AB=4, AD=5, \\angle A=60^{\\circ}$, $\\frac{BC}{CD}=$ $\\qquad$ .",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 96310,
        "difficulty": null
      },
      {
        "id": "8098033506eb6b8e5c548b1ad4f3787bafbf0d6c52f71b640c7ce492cee1bdcb",
        "domain": "numina",
        "question": "3. Can a certain number of circles, the sum of whose radii is 1999, be placed in a circle of radius 1 such that no two of them have common interior points?",
        "answer": "3998",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187908,
        "difficulty": null
      },
      {
        "id": "8400cb4629b51d97a59c8d9b6a329274c3034ff4639f5087837022ada471d21f",
        "domain": "numina",
        "question": "Problem 1. The sum of the first four terms of an arithmetic progression, as well as the sum of the first seven of its terms, are natural numbers. In addition, its first term $a_{1}$ satisfies the inequality $a_{1} \\leqslant \\frac{2}{3}$. What is the greatest value that $a_{1}$ can take?",
        "answer": "\\frac{9}{14}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9156,
        "difficulty": null
      },
      {
        "id": "c6f342783959205fa7c0bc21e3a87b2c61eb5341e99b073b8d5826365a2fbb68",
        "domain": "numina",
        "question": "8. In an acute-angled triangle $A B C$, a point $Q$ is chosen on side $A C$ such that $A Q: Q C=1: 2$. From point $Q$, perpendiculars $Q M$ and $Q K$ are dropped to sides $A B$ and $B C$ respectively. It is given that $B M: M A=4: 1, B K=K C$. Find $M K: A C$.",
        "answer": "\\frac{2}{\\sqrt{10}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8614,
        "difficulty": null
      },
      {
        "id": "550cd975437d831950d73ea3ec47d064504d1c4d2b26f37c319b3d6a99dd0d36",
        "domain": "numina",
        "question": "Problem 7.1. In the picture, nine small squares are drawn, with arrows on eight of them. The numbers 1 and 9 are already placed. Replace the letters in the remaining squares with numbers from 2 to 8 so that the arrows from the square with the number 1 point in the direction of the square with the number 2 (the number 2 can be in square $A$ or $B$), the arrows from the square with the number 2 point in the direction of the square with the number 3, and so on, the arrows from the square with the number 8 point in the direction of the square with the number 9.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_564c13f715a760703913g-18.jpg?height=389&width=379&top_left_y=96&top_left_x=529)\n\nConstruct the correspondence.\n\n- In square $A$\n- In square $B$\n- In square $C$\n- In square $D$\n- In square $E$\n- In square $F$\n- In square $G$\n- stands the number 2.\n- stands the number 3.\n- stands the number 4.\n- stands the number 5.\n- stands the number 6.\n- stands the number 7.\n- stands the number 8.",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16911,
        "difficulty": null
      },
      {
        "id": "4bd27757d08f18cb855f62bbd47585072686e326399563ce735c480d8763f0f0",
        "domain": "numina",
        "question": "A square-based right pyramid has its lateral faces forming a $60^{\\circ}$ angle with the base. The plane that bisects the angle between the base and one of the lateral faces divides the pyramid into two parts. Determine the ratio of the volumes of the two resulting solids.",
        "answer": "\\frac{3}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 167047,
        "difficulty": null
      },
      {
        "id": "0a97c469f52a1aaca5af13a90ca4d59cfe61a0d4f2f6e3f1dbf3d24e63b4d85b",
        "domain": "numina",
        "question": "14. In $\\triangle A B C$, $B D=D C$. There is a piece of cheese on side $A C$, located at the point closest to point $C$ among the four equal segments. On $A D$, there are three mirrors $W_{1}, W_{2}, W_{3}$, which divide $A D$ into four equal segments. A very suspicious mouse is crawling on $A B$ (from $A$ to $B$), with $A B=400$ meters. The mouse can observe the cheese when the mouse, a certain mirror, and the cheese are aligned. Due to its suspicion, the mouse hopes to see the cheese multiple times to ensure that the cheese has not been eaten by other mice before it reaches it. Therefore, in the first minute, it crawls forward 80 meters, in the second minute, it retreats 20 meters, in the third minute, it crawls forward 80 meters, in the fourth minute, it retreats 20 meters, and so on. When the mouse reaches point $B$, it directly rushes along $B C$ to eat the cheese. Question: How many times can the mouse see the cheese on segment $A B$?",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124818,
        "difficulty": null
      },
      {
        "id": "f34e2698c174931545ce723b37c9d3864e6e8658490ada6b7b8dbfe6543ee10f",
        "domain": "numina",
        "question": "Problem 1. At a round table, 60 people are sitting. Each of them is either a knight, who always tells the truth, or a liar, who always lies. Each person at the table said: \"Among the next 3 people sitting to my right, there is no more than one knight.\" How many knights could have been sitting at the table? List all possible options and prove that there are no others.",
        "answer": "30",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 4520,
        "difficulty": null
      },
      {
        "id": "28202d15a9384d3360b480992c9f283876dd38d935f5b002c0cf2908c1457636",
        "domain": "numina",
        "question": "2. (2 points) Point $M$ lies on the side of a regular hexagon with side length 12. Find the sum of the distances from point $M$ to the lines containing the other sides of the hexagon.",
        "answer": "36\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20936,
        "difficulty": null
      },
      {
        "id": "8d0c9eca497ac3a00e151b7d4d2b109f0a25842162354598ae3eef344ed73920",
        "domain": "numina",
        "question": "$11 \\cdot 42$ For an integer $x \\geqslant 1$, let $p(x)$ be the smallest prime that does not divide $x$, and $q(x)$ be the product of all primes less than $p(x)$. In particular, $p(1)=2$, and if there is some $x$ such that $p(x)=2$, then $q(x)=1$.\nThe sequence $x_{0}, x_{1}, x_{2}, \\cdots$ is defined by:\n$$\n\\begin{array}{c}\nx_{0}=1, \\\\\nx_{n+1}=\\frac{x_{n} p\\left(x_{n}\\right)}{q\\left(x_{n}\\right)}\n\\end{array}\n$$\n\nwhere $n \\geqslant 0$, find all integers $n$ such that $x_{n}=1995$.\n(36th International Mathematical Olympiad Preliminary Question, 1995)",
        "answer": "142",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 120301,
        "difficulty": null
      },
      {
        "id": "8509a7e9e88d291d89839aa1eae380e46c953db8d917a81d7dcdaa6dd527d20f",
        "domain": "numina",
        "question": "7. Given $P$ is the circumcenter of $\\triangle A B C$, and $\\overrightarrow{P A}+\\overrightarrow{P B}=\\lambda \\overrightarrow{P C}, \\tan C$ $=\\frac{12}{5}$, then the value of the real number $\\lambda$ is $\\qquad$.",
        "answer": "-\\frac{10}{13}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124055,
        "difficulty": null
      },
      {
        "id": "236552018af8cfa7ad629827f8f2e3c5bfdb7028f7ba686eb0bd3ca49d6dba63",
        "domain": "numina",
        "question": "C3 a) In how many ways can we read the word SARAJEVO from the table below, if it is allowed to jump from cell to an adjacent cell (by vertex or a side) cell?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_17e61ec1e039d21af3a0g-11.jpg?height=497&width=596&top_left_y=631&top_left_x=730)\n\nb) After the letter in one cell was deleted, only 525 ways to read the word SARAJEVO remained. Find all possible positions of that cell.",
        "answer": "750",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 259,
        "difficulty": null
      },
      {
        "id": "afb988d92ac86ed7f337349b4e93d6962c9788dca0f624e77d53fec7c445c994",
        "domain": "numina",
        "question": "2. Let positive numbers $a, b, c, x, y, z$ satisfy $c y + b z = a, a z + c x = b, b x + a y = c$, find the minimum value of the function $f(x, y, z)=$\n$\\frac{x^{2}}{1+x}+\\frac{y^{2}}{1+y}+\\frac{z^{2}}{1+z}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116888,
        "difficulty": null
      },
      {
        "id": "6298c831485ed3b863d085a4f170d30668846446a22739d8b63d5e6267aaca1c",
        "domain": "numina",
        "question": "3.10. In the square $A B C D$, an isosceles triangle $A E F$ is inscribed; point $E$ lies on side $B C$, point $F$ lies on side $C D$, and $A E=A F$. The tangent of angle $A E F$ is 3. Find the cosine of angle $F A D$.",
        "answer": "\\frac{2\\sqrt{5}}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 50169,
        "difficulty": null
      },
      {
        "id": "b7b3b1c89c22e78518d77204a936b828b1f2e184dbdb2046533a4f646c4996c9",
        "domain": "numina",
        "question": "2. Radmila brought a basket of eggs to the market. To the first buyer, she sold half of all the eggs and one more egg. To the second, she sold half of the remaining eggs and one more egg. Similarly, to the third, she sold half of the remaining eggs and one more egg. Then, one egg was left in the basket. How many eggs did Radmila bring to the market?",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 189112,
        "difficulty": null
      },
      {
        "id": "f10dd67547511b836a04cdafe42e0e5db1ab77a1c34548ae560107db477bb0bb",
        "domain": "numina",
        "question": "Problem 4. The maze for mice (see the figure) is a $5 \\times 5$ meter square, and the mice can only run along the paths. Two pieces of cheese (marked with crosses) are placed at two intersections. At another intersection, a mouse (marked with a circle) is sitting. She can smell the cheese, but she needs to run the same distance to both pieces, so she doesn't know which one to choose and sits thoughtfully in place.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_51d7eed368663d90d59bg-2.jpg?height=260&width=273&top_left_y=1152&top_left_x=1002)\n\na) Mark five more intersections where the mouse could sit thoughtfully (from where she would need to run the same distance to both pieces of cheese).\n\n[2 points]\n\nb) Come up with two intersections where you can place a piece of cheese so that there are as many suitable intersections for a thoughtful mouse as possible.\n\n[up to 5 points]",
        "answer": "26",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22744,
        "difficulty": null
      },
      {
        "id": "e08af410dec859178b99d02147dcf79bd38ffff54ce5bce08adef040767674ad",
        "domain": "numina",
        "question": "## T-3\n\nFind the smallest integer $b$ with the following property: For each way of colouring exactly $b$ squares of an $8 \\times 8$ chessboard green, one can place 7 bishops on 7 green squares so that no two bishops attack each other.\n\nRemark. Two bishops attack each other if they are on the same diagonal.\n\nAnswer. 41",
        "answer": "41",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 176883,
        "difficulty": null
      },
      {
        "id": "c637a4443c1195c9ecbb979cebd1dfc9d87cde8e9443bbfda77534b93f811f3d",
        "domain": "numina",
        "question": "6. Given a rectangle $A B C D$ with point $S$ being the intersection of the diagonals. On side $\\overline{A D}$, points $E$ and $F$ are chosen such that $|A E|=|E F|=|F D|$. Determine the ratio of the area of pentagon $E B S C F$ to the area of rectangle $A B C D$.",
        "answer": "\\frac{5}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86958,
        "difficulty": null
      },
      {
        "id": "176a1e001e47cf03ce210cb8542611fa5b8b38b70ef65fdb08eb452330bb8742",
        "domain": "numina",
        "question": "130. How many four-digit numbers can be formed from the digits $0,1,2,3,4,5$, if the numbers must be odd and digits may be repeated?",
        "answer": "540",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 35953,
        "difficulty": null
      },
      {
        "id": "d5a42302b70c8cc7500a0ea0f07c48b9b9c902157dd720ef1bdf51e9a14c73c8",
        "domain": "numina",
        "question": "4. The number $a_{n}$ is formed by writing down the first n consecutive natural numbers in sequence, for example, $a_{13}=12345678910111213$. Determine how many numbers divisible by 24 are among the numbers $a_{1}, a_{2}, \\ldots, a_{10000}$.",
        "answer": "834",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 76143,
        "difficulty": null
      },
      {
        "id": "ff39b4ebe56a2aab7c07a8fdac8f647f3bf018f598379fca29a9e85bfa1e1862",
        "domain": "numina",
        "question": "Task 4. Masha wrote a three-digit number on the board, and Vera wrote the same number next to it, but she swapped the last two digits. After that, Polina added the obtained numbers and got a four-digit sum, the first three digits of which are 195. What is the last digit of this sum? (The answer needs to be justified.)\n\nAnswer: 4.",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 16008,
        "difficulty": null
      },
      {
        "id": "a91fca70e9a0604ae303f851485c398bc9d8e567064b24e95320d546851775e0",
        "domain": "numina",
        "question": "## SUBJECT I\n\nDetermine the non-zero natural number n such that $\\frac{2}{3 \\cdot 5}+\\frac{2}{5 \\cdot 7}+\\ldots . .+\\frac{2}{(n-2) \\cdot n}=\\frac{332}{999}$.",
        "answer": "999",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178752,
        "difficulty": null
      },
      {
        "id": "88d68bd44bf70736a21541356df6a3412983213d577fadc652e2a238cac1ec83",
        "domain": "numina",
        "question": "【Question 19】\nThere are 2014 apples in three baskets. If the same number of apples is taken from each basket, the second basket will have 49 apples left, and the number of apples left in the first basket is twice the number of apples left in the third basket. How many apples were originally in the third basket? $\\qquad$",
        "answer": "655",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 100231,
        "difficulty": null
      },
      {
        "id": "4ab14c77f23dc3a376377c26b3ed914978db0339e70bbf618bd17e971eb13f04",
        "domain": "numina",
        "question": "Zhendarov R.G.\n\nFind all such prime numbers $p$ that the number $p^{2}+11$ has exactly six distinct divisors (including one and the number itself).",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69977,
        "difficulty": null
      },
      {
        "id": "5127a53bb7b9ac64399bcdd027ff7239c6d899520fcf1cee334302bb1e5b29b8",
        "domain": "numina",
        "question": "Problem 4.7. Vanya goes to the swimming pool every Wednesday and Friday. After one of his visits, he noticed that he had already gone 10 times this month. What will be the date of the third visit in the next month if he continues to go on Wednesdays and Fridays?",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 14235,
        "difficulty": null
      },
      {
        "id": "9c572dd9b3de62088288cb660e2756e70625e92090af91cbb89626e231df0eda",
        "domain": "numina",
        "question": "18. As shown in the figure, the lengths of the base edges of the rectangular prism are $1 \\mathrm{~cm}$ and $3 \\mathrm{~cm}$, and the height is $6 \\mathrm{~cm}$. If a thin string is used to start from point $A$, pass through the four sides and wrap around once to reach point $B$, then the shortest length of the string used is $\\qquad$ $\\mathrm{cm}$.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 103079,
        "difficulty": null
      },
      {
        "id": "ce0074b936aa3440bed9425149a71afea24dffe6ac89d407364565a348b121c8",
        "domain": "numina",
        "question": "## Task $1 / 88$\n\nLet $x$ be the sum of the digits of $1988^{8891}$, $y$ the sum of the digits of $x$, and $z$ the sum of the digits of $y$. Determine $z$!",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 182814,
        "difficulty": null
      },
      {
        "id": "d0115dbf8574d2e5f3035813fe26c2c7b090027acc0e2e505990741420b5d97b",
        "domain": "numina",
        "question": "1 In a certain municipality, they want to set up a hydroelectric power plant, and for this purpose, they wish to utilize the energy of the water flowing at the border. Preliminary calculations are to be made to determine how many light bulbs the water's energy can power in the case of the lowest water level. The river flows in a rectangular parallelogram-shaped channel at one point, which is $5 \\mathrm{~m}$ wide. Even at the lowest water level, the depth of the water is $0.9 \\mathrm{~m}$. The speed of the water was determined using a floating piece of wood and it was found that it covers a distance of $70 \\mathrm{~m}$ in 100 seconds. The terrain conditions are such that a drop of $3 \\mathrm{~m}$ can be easily established. The turbines and associated generators convert $80 \\%$ of the available energy into electrical energy, and $10 \\%$ of the energy is lost in the transmission lines. A 16 standard candlepower lamp in a carbon filament operates at 110 Volts and 0.5 Amperes.[^0]\n\n\n[^0]:    1* We also gladly accept solutions to starred problems from university students.",
        "answer": "1214",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 176306,
        "difficulty": null
      },
      {
        "id": "230a81e2eee97d7d3f44184e7b38e999d3a130dea87f26d00a5a4f2c5828272a",
        "domain": "numina",
        "question": "24. Mowgli asked his monkey friends to bring him some nuts. The monkeys gathered an equal number of nuts and carried them to Mowgli. But on the way, they quarreled, and each monkey threw a nut at every other monkey. As a result, Mowgli received only 35 nuts. How many nuts did the monkeys gather, if it is known that each of them brought more than one nut?",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 39422,
        "difficulty": null
      },
      {
        "id": "40f3f85318cf419dc64976db02438c9063c33d224cec2718e46debdc906a6e33",
        "domain": "numina",
        "question": "7.2. Losharik is going to visit Sovunya along the river at a speed of 4 km/h. Every half hour, he launches paper boats that float to Sovunya at a speed of 10 km/h. With what time interval do the boats arrive at Sovunya? (Provide a complete solution, not just the answer.)",
        "answer": "18",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9687,
        "difficulty": null
      },
      {
        "id": "5152204e72c8385c71381b1691402e43dc0429ac2a9eb4cd5e2c7750e1f846f1",
        "domain": "numina",
        "question": "98. There are 3 integers $\\mathrm{A}, \\mathrm{B}, \\mathrm{C}$, if 5 times $\\mathrm{B}$ is 4 times the difference between $\\mathrm{A}$ and 1, 5 times $\\mathrm{C}$ is 4 times the difference between $\\mathrm{B}$ and 1, then the smallest value of $\\mathrm{A}$ is $\\qquad$.",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109042,
        "difficulty": null
      },
      {
        "id": "16aa1e7c8937109a402b3e316d3b46c51ec72c5d917ed4f8ac36a7b9e65c74d9",
        "domain": "numina",
        "question": "For what values of $x$ is the expression $4^{\\sin ^{2} x}+8^{\\cos ^{2} x}$ minimal? What is the minimum value?",
        "answer": "5\\cdot\\sqrt[5]{\\frac{16}{27}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 166811,
        "difficulty": null
      },
      {
        "id": "490b4d4f3a1fccffd85f0f788a3dbc1408f743ce59e35d6dc1589808398f107f",
        "domain": "numina",
        "question": "Task A-3.5. (4 points)\n\nIf the lengths of the sides of a triangle satisfy the equation $\\frac{a+b}{b+c}=\\frac{c}{a-b}$, determine the measure of the largest angle of this triangle.",
        "answer": "120",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87474,
        "difficulty": null
      },
      {
        "id": "87751476e3ab8542ced60f9bebf4063a77f68d8698b706c4a7ba76070f5083d8",
        "domain": "numina",
        "question": "In the diagram, points $B$ and $C$ lie on $A D$ and point $E$ lies on $C F$. The measures of four angles are shown. What is the value of $x+y+z$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2efeb2001664c4e9869eg-1.jpg?height=429&width=656&top_left_y=1268&top_left_x=797)",
        "answer": "124",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194287,
        "difficulty": null
      },
      {
        "id": "dad9aa50b7392688fd9bc06d5962f3464c359af8a3cc2660ede645782484e212",
        "domain": "numina",
        "question": "[ Systems of points and segments (other) ] [ Classical combinatorics (other) ]\n\nOn a circle, 20 points are marked. How many triples of chords with endpoints at these points exist such that each chord intersects the other two (possibly at the endpoints)?",
        "answer": "156180",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 62493,
        "difficulty": null
      },
      {
        "id": "681503fd610a8d7f659698f28d93409626702b1a854f25cab24866186e03e0c6",
        "domain": "numina",
        "question": "# PROBLEM 1\n\na) Show that: $A=\\sqrt{1+3+5+\\ldots+2015} \\in Q$\n\nb) If the real numbers $a$ and $b$ satisfy the relation: $a^{2}+b^{2}-4 \\sqrt{3} a-6 \\sqrt{2} b+30=0$, then calculate:\n\n$$\n\\mathrm{E}=\\left(2 \\mathrm{a}^{-1}+3 \\mathrm{~b}^{-1}\\right)\\left(\\frac{1}{b^{-1}}-\\frac{1}{a^{-1}}\\right)\n$$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179006,
        "difficulty": null
      },
      {
        "id": "424ae34735d1fbac809702bc8d83ff56828abf47c3abfa887ab75fd66e7bfb13",
        "domain": "numina",
        "question": "10.1. The first term of the sequence is 934. Each subsequent term is equal to the sum of the digits of the previous term, multiplied by 13. Find the 2013-th term of the sequence.",
        "answer": "130",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15681,
        "difficulty": null
      },
      {
        "id": "8abf89c3964a80e5ed4ba5df132fad8bb786da216140b8d421a9f142a3f4c71b",
        "domain": "numina",
        "question": "We consider all positive real numbers $a, b, c, d$ for which $(a+c)(b+d)=a c+b d$ holds.\nDetermine the smallest value that $S=\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}$ can take.",
        "answer": "8",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 195931,
        "difficulty": null
      },
      {
        "id": "5829b40270862a064931a989fef39cc2b8f9281f1fdfde22c9efa94dafb8137d",
        "domain": "numina",
        "question": "1. Option 1. To a confectionery factory for cake packaging, 5 rolls of ribbon, each 60 m long, were delivered. How many cuts need to be made to get pieces of ribbon, each 1 m 50 cm long?",
        "answer": "195",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 16270,
        "difficulty": null
      },
      {
        "id": "b78c06bdc1479f0a1aebbb3639167ffc40f0f698694a68c2fdd5d326d368eb9e",
        "domain": "numina",
        "question": "97.3. Let $A, B, C$, and $D$ be four different points in the plane. Three of the line segments $A B, A C, A D, B C, B D$, and $C D$ have length a. The other three have length $b$, where $b>a$. Determine all possible values of the quotient $\\frac{b}{a}$.",
        "answer": "\\frac{\\sqrt{5}+1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 401,
        "difficulty": null
      },
      {
        "id": "830ee55c33e1599051e28f74f3999199e8f77b0fcc4d2c321e021f553f16b75e",
        "domain": "numina",
        "question": "[Mathematical logic (miscellaneous).] Case analysis $\\quad]$\n\nAfter a hockey game, Anton said that he scored 3 goals, and Ilya only one. Ilya said that he scored 4 goals, and Seryozha as many as 5. Seryozha said that he scored 6 goals, and Anton only two. Could it be that together they scored 10 goals, given that each of them told the truth once and lied once?",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 53491,
        "difficulty": null
      },
      {
        "id": "6a563151b05c7846216699243622cdc9513dcad270fbaa13876d6c33149d03b9",
        "domain": "numina",
        "question": "13.4. 19 ** In $\\triangle A B C$ with a fixed perimeter, it is known that $|A B|=6$, and when vertex $C$ is at a fixed point $P$, $\\cos C$ has a minimum value of $\\frac{7}{25}$.\n（1）Establish an appropriate coordinate system and find the equation of the locus of vertex $C$;\n(2) Draw a line through point $A$ that intersects the curve from (1) at points $M$ and $N$, and find the minimum value of $|\\overrightarrow{B M}| \\cdot|\\overrightarrow{B N}|$.",
        "answer": "16",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129191,
        "difficulty": null
      },
      {
        "id": "707e230601cb47575c63a26697ae4bc6035e43696b271c844bba746d15ac1abf",
        "domain": "numina",
        "question": "Problem 5.7. In a class, there are 31 students. Three of them have exactly three friends each, the next three have six each, the next three have nine each, ..., and the next three have thirty each. How many friends does the 31st student have? (Friendship between people is mutual.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_32e5791ee8081231aabeg-6.jpg?height=494&width=525&top_left_y=80&top_left_x=464)\n\nFig. 2: to the solution of problem 5.6\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_32e5791ee8081231aabeg-6.jpg?height=260&width=521&top_left_y=661&top_left_x=466)\n\nFig. 3: to the solution of problem 5.6\n\n#",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13594,
        "difficulty": null
      },
      {
        "id": "31288b6a4784d0575734afb76144d354f4c60e1af998c919473713c93726b8fc",
        "domain": "numina",
        "question": "4. Arrange the digits $1,2,3,4,5,6,7,8$ to form two 4-digit integers whose difference is as small as possible.\nExplain clearly why your arrangement achieves the smallest possible difference.",
        "answer": "247",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 154944,
        "difficulty": null
      },
      {
        "id": "02d9692616bbf74c24b15813a8a855edc3519e5f891256e3f604e1791f966b92",
        "domain": "numina",
        "question": "In triangle $ABC$, side $BC$ is equal to 4, and the median drawn to this side is equal to 3. Find the length of the common chord of two circles, each of which passes through point $A$ and is tangent to $BC$, with one touching $BC$ at point $B$ and the other at point $C$.",
        "answer": "\\frac{5}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58876,
        "difficulty": null
      },
      {
        "id": "6e5175a7de1854314767b52f35e95b92d35a383db34d98d530aeec1f7f569146",
        "domain": "numina",
        "question": "12. (10 points) Xiaoming went to a store with 100 RMB to buy stationery. After returning home, he counted the RMB he got back, which included 4 different denominations of banknotes and 4 different denominations of coins. The banknotes were all worth more than 1 yuan, and the coins were all worth less than 1 yuan. Additionally, the sum of the values of all banknotes, in yuan, is divisible by 3, and the sum of the values of all coins, in fen, is divisible by 7. How much money did Xiaoming spend at most? (Note: The store has banknotes of 100 yuan, 50 yuan, 20 yuan, 10 yuan, 5 yuan, and 1 yuan, and coins of 5 jiao, 1 jiao, 5 fen, 2 fen, and 1 fen for change)",
        "answer": "63",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 139750,
        "difficulty": null
      },
      {
        "id": "f7cece2179d0b241bcbc1df842134532fc7e26cb381a8453a731444b5bd6d641",
        "domain": "numina",
        "question": "## Task 2\n\nSubtract from 3182100 the product of the numbers 56823 and 56. By how much is the difference greater than 10?",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 186824,
        "difficulty": null
      },
      {
        "id": "9dc3c698a08a37995d1f0050472eb17fef86f440b564937699d8a06a0f33a65b",
        "domain": "numina",
        "question": "3. A three-digit number, where the hundreds and units digits are the same, and this three-digit number is a perfect square. There are $\\qquad$ such three-digit numbers.\n\nNote: A perfect square can be expressed in the form of $a \\times a$, such as $9=3 \\times 3, 25=5 \\times 5$, both 9 and 25 are perfect squares.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 121181,
        "difficulty": null
      },
      {
        "id": "2565fdd633ee1e21412e8b3fe321e825205de9b0b5eec8802f29b84d3cb9a423",
        "domain": "numina",
        "question": "3. Find the number of distinct reduced quadratic trinomials (i.e., with the leading coefficient equal to 1) with integer coefficients such that they have at least one root, all their roots are powers of the number 3 with integer non-negative exponents, and their coefficients in absolute value do not exceed \\(27^{47}\\).",
        "answer": "5111",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3510,
        "difficulty": null
      },
      {
        "id": "202338443d1e8682fd0c994eb6969a7d0a9f691144b79540478567732db73300",
        "domain": "numina",
        "question": "3. [7 points] Let $M$ be a figure on the Cartesian plane consisting of all points $(x, y)$ such that there exists a pair of real numbers $a, b$ for which the system of inequalities is satisfied\n\n$$\n\\left\\{\\begin{array}{l}\n(x-a)^{2}+(y-b)^{2} \\leqslant 5 \\\\\na^{2}+b^{2} \\leqslant \\min (4 a-2 b ; 5)\n\\end{array}\\right.\n$$\n\nFind the area of the figure $M$.",
        "answer": "15\\pi-\\frac{5\\sqrt{3}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 3356,
        "difficulty": null
      },
      {
        "id": "0341510bcf16e43d169543ae957cc30f905b535026b65166dce56f3f2130ecb9",
        "domain": "numina",
        "question": "Paul has 6 boxes, each of which contains 12 trays. Paul also has 4 extra trays. If each tray can hold 8 apples, what is the largest possible number of apples that can be held by the 6 boxes and 4 extra trays?",
        "answer": "608",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 193986,
        "difficulty": null
      },
      {
        "id": "567d9321ee7c55ee1a15655843a1f5d4c194a64bf536ae05dfc820ff145d7208",
        "domain": "numina",
        "question": "Example 3 Find all positive integers $k$ such that the indeterminate equation $x^{2}+y^{2}=k x y-1$ has positive integer solutions for $x$ and $y$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 96270,
        "difficulty": null
      },
      {
        "id": "d00cd3b4fc64ef1751fe0592168562942d4ffe2ccac8c394c3b8c0171b353d8b",
        "domain": "numina",
        "question": "3. Given three points $A, B, C$, forming a triangle with angles $30^{\\circ}, 45^{\\circ}, 105^{\\circ}$. Two of these points are chosen, and the perpendicular bisector of the segment connecting them is drawn, after which the third point is reflected across this perpendicular bisector. This results in a fourth point $D$. With the resulting set of 4 points, the same procedure is carried out - two points are chosen, the perpendicular bisector is drawn, and all points are reflected across it. What is the maximum number of distinct points that can be obtained by repeatedly performing this procedure?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20275,
        "difficulty": null
      },
      {
        "id": "a15d2ef73c0c76d4c0903458dedd665825b2371cfe7f1c126445c7cf06f006e1",
        "domain": "numina",
        "question": "(2) Given the function $f(x)=\\sqrt{3} \\sin 2 x+2 \\cos ^{2} x+a$, if the minimum value of $f(x)$ on the interval $\\left[0, \\frac{\\pi}{2}\\right]$ is -1, then the value of $a$ is $\\qquad$.",
        "answer": "-1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 115430,
        "difficulty": null
      },
      {
        "id": "ac8f8362ee2e7b23cbbfe8794f32560dc85fb15ed0a48a58df8a9b83db2e96fc",
        "domain": "numina",
        "question": "How many positive five-digit whole numbers are there for which the product of their five digits is 900?\n\n(Karl Czakler)\n\nAnswer. 210",
        "answer": "210",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 190975,
        "difficulty": null
      },
      {
        "id": "d2277bff2a85b41842856dff4628897ca1df1bb0e4c30b4c356d58523ef7bec5",
        "domain": "numina",
        "question": "Problem 3. In the morning, a dandelion blooms, it flowers yellow for two days, on the third day in the morning it turns white, and by evening it sheds its seeds. Yesterday afternoon, there were 20 yellow and 14 white dandelions on the meadow, and today there are 15 yellow and 11 white.\na) How many yellow dandelions were there on the meadow the day before yesterday\n\nb) How many white dandelions will there be on the meadow tomorrow? [6 points] (",
        "answer": "25",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22793,
        "difficulty": null
      },
      {
        "id": "607b3a772b17b0df61ef9c5ff3e1597ae5ae90e7676529751883f5e20722bb08",
        "domain": "numina",
        "question": "10. (5 points) According to the pattern shown in the figure, deduce that $M=$\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "1692",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 129392,
        "difficulty": null
      },
      {
        "id": "0e8214c4e24c85bb2ded3764d9534b82eed3f73eea3f006dc4fe271b1bbebfb0",
        "domain": "numina",
        "question": "11.4. 2011 Warehouses are connected by roads in such a way that from any warehouse you can drive to any other, possibly by driving along several roads. On the warehouses, there are $x_{1}, \\ldots, x_{2011}$ kg of cement respectively. In one trip, you can transport an arbitrary amount of cement from any warehouse to another warehouse along the road connecting them. In the end, according to the plan, there should be $y_{1}, \\ldots, y_{2011}$ kg of cement on the warehouses respectively, with\n\n$$\nx_{1}+x_{2}+\\ldots+x_{2011}=y_{1}+y_{2}+\\ldots+y_{2011}\n$$\n\nWhat is the minimum number of trips required to fulfill the plan for any values of the numbers $x_{i}$ and $y_{i}$ and any road network?\n\n(P. Karasev)",
        "answer": "2010",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12520,
        "difficulty": null
      },
      {
        "id": "3081c70d733ff43daba552a06d794d7b06ce7982b9e062d12d13c5f926aae38e",
        "domain": "numina",
        "question": "[The ratio in which the bisector divides the side.] Law of Cosines\n\nIn triangle $ABC$, the bisector of angle $BAC$ intersects side $BC$ at point $M$. It is known that $AB = BC = 2AC$, $AM = 4$. Find the area of triangle $ABC$.",
        "answer": "\\frac{18\\sqrt{15}}{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59830,
        "difficulty": null
      },
      {
        "id": "4709862e881a139fb11deee1c81b451776ef6639bedd4a56c71fda95f7a996cf",
        "domain": "numina",
        "question": "【Question 21】\nFour football teams participate in a round-robin tournament, where each pair of teams plays one match. The winning team gets 3 points, the losing team gets 0 points, and in the case of a draw, both teams get 1 point. It is known that after all the matches, the points of the four teams are four consecutive positive integers. Find the product of these four numbers.",
        "answer": "120",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 105754,
        "difficulty": null
      },
      {
        "id": "a1f279998f8acd7aab9dfc2ec96362f73631ffa3c82c6e259bbd37bc4cf10164",
        "domain": "numina",
        "question": "8.6. A triangle is divided into triangular cells as shown in the figure. A natural number is written in each cell. For each side of the triangle, there are four layers parallel to this side, containing seven, five, three, and one cell, respectively. It turns out that the sum of the numbers in each of these twelve layers is a prime number. What is the smallest possible sum of all the written numbers?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6ab074edf53f8bb32518g-2.jpg?height=334&width=391&top_left_y=1843&top_left_x=1546)",
        "answer": "22",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 15447,
        "difficulty": null
      },
      {
        "id": "c9b4f26dec89614d7bb2292007e8f19a7dd28b96f57dc2fea1e76abdbb428d15",
        "domain": "numina",
        "question": "9.6. Find the largest $n$ such that the sum of the fourth powers of any $n$ prime numbers greater than 10 is divisible by $n$.",
        "answer": "240",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 17283,
        "difficulty": null
      },
      {
        "id": "9cce4816219dda953fe45940ca5443abb24a6ab39883f0e86e97e195036d5554",
        "domain": "numina",
        "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d43b1f0a7bd77fffa87ag-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
        "answer": "43",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17399,
        "difficulty": null
      },
      {
        "id": "5257e7844979896b31750e3056ac11c32a50d4f2edf35cdc712b8d67aff9a964",
        "domain": "numina",
        "question": "## Task B-1.1.\n\nFor real numbers $x$ and $y$, if $x-y=6$ and $x^{2}+y^{2}=22$, what is $x^{3}-y^{3}$?",
        "answer": "90",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88715,
        "difficulty": null
      },
      {
        "id": "347c43224b2590db8dff6924cf7a28d995f76b1e9afb32f9b2b97039bd226aa3",
        "domain": "numina",
        "question": "1. Calculate the value of the expression:\n\n$$\n\\frac{(\\sqrt{666}+\\sqrt{888})^{2}-\\sqrt{666^{2}+888^{2}}}{444}\n$$",
        "answer": "1+2\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86594,
        "difficulty": null
      },
      {
        "id": "26db48df0aa2ab094f1fec0e21a9f87472c8d90e16e9fa00dc4790a06982e673",
        "domain": "numina",
        "question": "3-ча 1. Determine the coefficients that will stand by $x^{17}$ and $x^{18}$ after expanding the brackets and combining like terms in the expression\n\n$$\n\\left(1+x^{5}+x^{7}\\right)^{20}\n$$",
        "answer": "3420",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 26253,
        "difficulty": null
      },
      {
        "id": "8e27017663ab60a319c089fb20a753caf01d5ae83a001b2dd6dce2d4f8d45899",
        "domain": "numina",
        "question": "9. (10 points) In the figure, $AB$ is the diameter of circle $O$, 6 cm long, and square $BCDE$ has one vertex $E$ on the circumference of the circle, $\\angle ABE=45^{\\circ}$. The difference between the area of the non-shaded part of circle $O$ and the area of the non-shaded part of square $BCDE$ is $\\qquad$ square centimeters (take $\\pi=3.14$)",
        "answer": "10.26",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105666,
        "difficulty": null
      },
      {
        "id": "4724f62b0034f66d8d55ede61204b68de14aa291b98179b2e46374a7912cd63f",
        "domain": "numina",
        "question": "4. It is known that $3 \\operatorname{tg}^{2} \\gamma+3 \\operatorname{ctg}^{2} \\gamma+\\frac{2}{\\sin ^{2} \\gamma}+\\frac{2}{\\cos ^{2} \\gamma}=19$. Find the value of the expression $\\sin ^{4} \\gamma-\\sin ^{2} \\gamma$.\n\nAnswer. $-\\frac{1}{5}$.",
        "answer": "-\\frac{1}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 22959,
        "difficulty": null
      },
      {
        "id": "6bd9595666ea3ed2230b2eb212234c2819231586934fd7da6d78d029d8ce4217",
        "domain": "numina",
        "question": "The last digit of each number is 6. What is the condition for the product of the two numbers to end in 36?\n\n翻译完成，但根据您的要求，我保留了源文本的换行和格式。请注意，这里提供的翻译是将匈牙利语翻译成了英文。如果您需要进一步的帮助或有其他要求，请告诉我。",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 172580,
        "difficulty": null
      },
      {
        "id": "276d0f90150353366c9d11d655a7fc7490604b66d9a4756f508c5382d1b56bf8",
        "domain": "numina",
        "question": "## Task 1 - 291211\n\nDetermine the number of all natural numbers $z$ with the following properties:\n\n(1) The decimal representation of $z$ consists of five pairwise distinct digits.\n\n(2) The first and the last digit in it are different from 0.\n\n(3) If $z^{\\prime}$ is the number whose digit representation is obtained from that of $z$ by reversing the order, then the digit representation of the number $z+z^{\\prime}$ consists of all identical digits.",
        "answer": "56",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 186173,
        "difficulty": null
      },
      {
        "id": "52905208abbffb8b231930d184d1a9da595b467fb3a3639ad7ba74b9afdbdda7",
        "domain": "numina",
        "question": "Example 2.2. $I=\\int_{0}^{2}(x-1)^{3} d x$.",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30845,
        "difficulty": null
      },
      {
        "id": "9f98939b6340fd129822e967df1a8c4b6d5731e9cc27b945bbcf9b936ab27a77",
        "domain": "numina",
        "question": "Variant 11.3.3. The polynomial $G(x)$ with real coefficients takes the value 2022 at exactly five different points $x_{1}<x_{2}<x_{3}<x_{4}<x_{5}$. It is known that the graph of the function $y=G(x)$ is symmetric with respect to the line $x=-7$.\n\n(a) (2 points) Find $x_{1}+x_{3}+x_{5}$.\n\n(b) (2 points) What is the smallest degree that $G(x)$ can have?",
        "answer": "-21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12191,
        "difficulty": null
      },
      {
        "id": "0d1910231f9e90b731c8f1ed7bc869850a714e36312e8153787deb3f6e14a1d0",
        "domain": "numina",
        "question": "3. A natural number greater than 0, if the sum of all its divisors equals twice itself, such a number is called a perfect number or a complete number. For example, the divisors of 6 are $1,2,3,6$, $1+2+3+6=12$, so 6 is the smallest perfect number. Whether there are infinitely many perfect numbers remains one of the challenges that have puzzled humans. Research on perfect numbers can start with calculating the sum of all divisors of a natural number, the sum of all divisors of 8128 is.",
        "answer": "16256",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 142496,
        "difficulty": null
      },
      {
        "id": "a9b5e23046f2e025bd36ddf125c0456ce55f1cf03bfb0c406efb7c64f06d9cd6",
        "domain": "numina",
        "question": "9. There is a heads up coin on every integer of the number line. Lucky is initially standing on the zero point of the number line facing in the positive direction. Lucky performs the following procedure: he looks at the coin (or lack thereof) underneath him, and then,\n- If the coin is heads up, Lucky flips it to tails up, turns around, and steps forward a distance of one unit.\n- If the coin is tails up, Lucky picks up the coin and steps forward a distance of one unit facing the same direction.\n- If there is no coin, Lucky places a coin heads up underneath him and steps forward a distance of one unit facing the same direction.\nHe repeats this procedure until there are 20 coins anywhere that are tails up. How many times has Lucky performed the procedure when the process stops?",
        "answer": "6098",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 151503,
        "difficulty": null
      },
      {
        "id": "ad3d402d2f9baccef3386369bcf4e77d1237a816e1438e9dbf2a982c8c6dbe42",
        "domain": "numina",
        "question": "2. Given two linear functions $f(x)$ and $g(x)$ such that the graphs $y=f(x)$ and $y=g(x)$ are parallel lines, not parallel to the coordinate axes. Find the minimum value of the function $(g(x))^{2}+$ $f(x)$, if the minimum value of the function $(f(x))^{2}+g(x)$ is -6.",
        "answer": "\\frac{11}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3397,
        "difficulty": null
      },
      {
        "id": "135d77c6f9771e403d4266f41230009dfd17b06ddab02936db900f5eff137d43",
        "domain": "numina",
        "question": "1. How many four-digit natural numbers can be written using the digits $0,1,2,3,4,5,6$, if all digits must be different and the sum of the digits must be $15?$",
        "answer": "66",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 161157,
        "difficulty": null
      },
      {
        "id": "98a614bc63649d4d96403e15940a2f27c7022769b5580cd5b87abca4781e11df",
        "domain": "numina",
        "question": "Problem 3. In a competition 8 judges marked the contestants by yes or no. It is known that for any two contestants, two judges gave both a yes; two judges gave the first one a yes and the second one a no; two judges gave the first one a no and the second one a yes, and finally, two judges gave both a no. What is the greatest possible number of contestants?\n\nEmil Kolev",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 176896,
        "difficulty": null
      },
      {
        "id": "654b1c2b1699499e594d47a7c90b45e1f32e7d87550f6548457122eca54ca2dc",
        "domain": "numina",
        "question": "25.15. Calculate $\\lim _{n \\rightarrow \\infty}(\\sqrt{n+1}-\\sqrt{n})$.",
        "answer": "0",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 38767,
        "difficulty": null
      },
      {
        "id": "8355ccea6cee062244e897d1891fccd1f89552bbf011848efbfaba576231b213",
        "domain": "numina",
        "question": "【Question 7】\nA car has a fuel tank capacity of 50 liters, and it departs from Shanghai to Harbin, which is 2560 kilometers away, with a full tank of fuel. It is known that the car consumes 8 liters of fuel for every 100 kilometers driven, and to ensure driving safety, at least 6 liters of fuel should be kept in the tank. Therefore, during the journey to Harbin, the car needs to refuel at least $\\qquad$ times.",
        "answer": "4",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 120047,
        "difficulty": null
      },
      {
        "id": "024e9ba2c4eb18cf051b67b3e69a9b22ea0b9cfcb6bb29c41b9b0f27696c160d",
        "domain": "numina",
        "question": "13. (15 points) As shown in the figure, in the isosceles right triangle $A B C$, $\\angle A=90^{\\circ}, A B=A C=1$, rectangle $E H G F$ is inside triangle $A B C$, and $G, H$ are on side $B C$. Find the maximum area of rectangle $E H G F$.",
        "answer": "\\frac{1}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 97970,
        "difficulty": null
      },
      {
        "id": "2ff67bc6a53a8abc6589ab0189be2d9e09f11410c6b9e1b70159e5b86dcf17b1",
        "domain": "numina",
        "question": "Multiplied several natural numbers and got 224, and the smallest number was exactly half of the largest. How many numbers were multiplied\n\n#",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 53279,
        "difficulty": null
      },
      {
        "id": "4566bb7e696a22c2f75d9508bbf62c7e5e0475f0897fb3ab3f4c6f5cb00a3e17",
        "domain": "numina",
        "question": "We roll a die 30 times. Determine the probability, to 8 decimal places, that among the numbers rolled there will be a six, and that before the first six, neither a five nor a four appears.",
        "answer": "0.33333333",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 167895,
        "difficulty": null
      },
      {
        "id": "20029a6e89996a0f34c6ba75ad2da0a967de2cc75e175a3171a281129bbaaec6",
        "domain": "numina",
        "question": "Example 1. Using the method of tangents, find the real root of the equation $x^{3}+x-3=0$.",
        "answer": "1.213412",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32866,
        "difficulty": null
      },
      {
        "id": "2b77b64f8484df68b702a49892dc68bcc2ef5ccc478b6554938bb5a1468c3270",
        "domain": "numina",
        "question": "Example 4-15 The 6 faces of a die have $1,2,3,4,5,6$ dots respectively, how many different schemes are there?",
        "answer": "30",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 112794,
        "difficulty": null
      },
      {
        "id": "2d7b8a349d46df3fff7b7a5c27f89b53c8c0a55a980c8efbc0fe39eabaefee62",
        "domain": "numina",
        "question": "B1. Find all real numbers $x$ that satisfy the equation\n\n$$\n\\log _{2}(10 x)+\\log _{4}(100 x)+\\log _{8}(1000 x)-2 \\log _{64} x=9\n$$\n\nWrite the result in the form of a simplified fraction.\n\n$$\n\\text { (6 points) }\n$$",
        "answer": "\\frac{16}{25}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163165,
        "difficulty": null
      },
      {
        "id": "ade18027a35d845dbcd33ac9074ca109e751b8a4b48fea0dccc6a7bd9dcb0da6",
        "domain": "numina",
        "question": "Task 1. Calculate the value of the numerical expression:\n\n$$\n\\frac{1}{17} \\cdot\\left(\\frac{2 \\frac{1}{2}+3 \\frac{1}{3}}{3 \\frac{1}{2}-2 \\frac{1}{3}}: \\frac{5 \\frac{3}{5}+1 \\frac{1}{3}}{6 \\frac{3}{5}-4 \\frac{1}{3}}\\right) \\cdot\\left(\\frac{\\frac{1}{4}-\\frac{1}{5}}{\\frac{1}{7}-\\frac{1}{8}}-\\frac{1}{5}\\right)\n$$",
        "answer": "\\frac{1}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188807,
        "difficulty": null
      },
      {
        "id": "48316094be59bd958ba75b2cfa6b42179b3b408ba206b7c62879928510dfb885",
        "domain": "numina",
        "question": "'1-188 Write the numbers $1,2,3, \\cdots, 1986,1987$ on the blackboard. At each step, determine some of the numbers written and erase them, replacing them with the remainder of their sum divided by 7. After several steps, two numbers remain on the blackboard, one of which is 987. What is the second remaining number?",
        "answer": "0",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 120559,
        "difficulty": null
      },
      {
        "id": "3e77a35ecbdc822d63255ab7465a20caaa6d3adc4d7c6f15f37a2fa56f3cfed8",
        "domain": "numina",
        "question": "[ The ratio of the areas of triangles with a common base or common height ]\n\nIn one island tribe, there is a custom - during a ritual dance, the shaman tosses three thin straight rods of the same length, tied in a shape resembling the letter П, high into the air. Adjacent rods are connected by a short thread and can therefore rotate freely relative to each other. The rods fall onto the sand, forming a random figure. If a self-intersection occurs (the first and third rods cross each other), the tribe will face poor harvests and various troubles in the coming year. If there is no self-intersection, the year will be successful - bountiful and happy. Find the probability that the rods will predict good fortune for the year 2017.",
        "answer": "\\frac{11}{12}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 55071,
        "difficulty": null
      },
      {
        "id": "1896b5454b892cc7524b1f55986197416e4d9b6ff170039dfdb508498a0acb41",
        "domain": "numina",
        "question": "# 7.2. (7 points)\n\nTwo pedestrians set out at dawn. Each walked at a constant speed. One walked from $A$ to $B$, the other from $B$ to $A$. They met at noon and, without stopping, arrived: one - in $B$ at 4 PM, and the other - in $A$ at 9 PM. At what hour was dawn that day?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14989,
        "difficulty": null
      },
      {
        "id": "8479ee690cfd0e65b2fe26c4f6968bec9d883bdfa60fbbddf53579a95b385fa4",
        "domain": "numina",
        "question": "9.4. In a class, there are 18 children. The parents decided to gift the children from this class a cake. To do this, they first asked each child the area of the piece they wanted to receive. After that, they ordered a square cake, the area of which is exactly equal to the sum of the 18 named numbers. However, upon seeing the cake, the children wanted their pieces to be square as well. The parents can cut the cake with cuts parallel to the sides of the cake (cuts are not required to start or end on the side of the cake). For what largest $k$ can the parents guarantee to cut out $k$ square pieces from the ordered cake, which can be given to $k$ children, so that each of them gets what they want?\n\n(A. Ibragimov, I. Bogdanov)",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17055,
        "difficulty": null
      },
      {
        "id": "949d74e6eef0da7570fa95ac5475516c57b088534454d59e3676abc4638d766e",
        "domain": "numina",
        "question": "156. Given a cube $A B C D A_{1} B_{1} C_{1} D_{1}$. Points $M$ and $N$ are taken on segments $A A_{1}$ and $B C_{1}$ such that the line $M N$ intersects the line $B_{1} D$. Find\n\n$$\n\\frac{\\left|B C_{1}\\right|}{|B N|}-\\frac{|A M|}{\\left|A A_{1}\\right|}\n$$",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29862,
        "difficulty": null
      },
      {
        "id": "f277ac19e92a2f92febba599158d743f44343e2fb3ee29a035376ea4eeb91745",
        "domain": "numina",
        "question": "6. (3 points) Given the cryptarithm: ЖАЛО + ЛОЖА = ОСЕНЬ. Identical letters represent identical digits, different letters represent different digits. Find the value of the letter А.",
        "answer": "8",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 20753,
        "difficulty": null
      },
      {
        "id": "984cebeca705ec2fda62010239700e00a790a5f2c2c85943328142cf23a801cc",
        "domain": "numina",
        "question": "30.5. Let each of the characters $A, B, C, D, E$ denote a single digit, and $A B C D E 4$ and $4 A B C D E$ represent six-digit numbers. If\n$$\n4 \\times A B C D E 4=4 A B C D E,\n$$\nwhat is $C$ ?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154070,
        "difficulty": null
      },
      {
        "id": "11ee7a2e4ab1f88e4fc6ba63985be718d4d115bd9c9aff39e83e55e2e0ebee15",
        "domain": "numina",
        "question": "3. Given that quadrilateral $A B C D$ is an isosceles trapezoid, $A B / / C D, A B=6, C D=16 . \\triangle A C E$ is a right triangle, $\\angle A E C=90^{\\circ}, C E=B C=A D$. Then the length of $A E$ is $\\qquad$ .",
        "answer": "4\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 105975,
        "difficulty": null
      },
      {
        "id": "557c46701c92a05dc4b3a6e8fd47e44c571cb123ffcdb5261b2b7188f3523367",
        "domain": "numina",
        "question": "11. A student missed the multiplication sign written between two three-digit numbers, mistaking them for a six-digit number, which turned out to be exactly 7 times the original product. What is the sum of these two three-digit numbers?",
        "answer": "286",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 117292,
        "difficulty": null
      },
      {
        "id": "06875baa0c1ae4b4ed58fec1ae54e1c140e8c66b7ec9f943199c117a2b5c7843",
        "domain": "numina",
        "question": "Task B-2.7. Today is Valentine's Day, and Valentino wants to go out with his girlfriend Lorna. With his 120 kuna, he can buy several roses and two lemonades. The price of one rose is equal to the price of one lemonade. Since they have a math test tomorrow, they will postpone the outing until Saturday. On Saturday, the price of one rose will decrease by 7 kuna. Valentino wants to buy the same number of roses, and instead of lemonades, he wants to buy a jumbo pizza and a Coca-Cola, which together cost 78 kuna. However, he would be short of 6 kuna. How many roses will Valentino buy for Lorna? At what price will he buy the roses on Saturday?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 88890,
        "difficulty": null
      },
      {
        "id": "013d442c6d220fee9b1f7b3e86901d82ebe87e21d45115616649efc39492f098",
        "domain": "numina",
        "question": "117. Given a regular tetrahedron with edge $a$. A sphere touches the three edges of the tetrahedron emanating from one vertex at their ends. Find the area of the part of the spherical surface located inside the tetrahedron.",
        "answer": "\\frac{\\pi^{2}}{6}(2\\sqrt{3}-3)",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29826,
        "difficulty": null
      },
      {
        "id": "fc8e5011597a1f9f1eced547c6caa2a3eacc1d7a463b004b8c887f9b9f942997",
        "domain": "numina",
        "question": "$\\underline{\\text { Folklore }}$\n\nIn a herd consisting of horses, two-humped and one-humped camels, there are a total of 200 humps.\n\nHow many animals are in the herd if the number of horses is equal to the number of two-humped camels? .",
        "answer": "200",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 68994,
        "difficulty": null
      },
      {
        "id": "89b0042f93cfcae800261e1f3b616b546fbad50cadb50a81f50889a5e7d704b8",
        "domain": "numina",
        "question": "Matěj and his friends went caroling. In addition to apples, walnuts, and gingerbread, each of the boys also received an orange. Jarda got one orange, Milan too. Radek, Patrik, Michal, and Dušan each got two oranges. Matěj got as many as four oranges, which was the most of all the boys. The other boys each got three oranges. How many boys went caroling if they received a total of 23 oranges?\n\n(M. Volfová)",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76530,
        "difficulty": null
      },
      {
        "id": "f8c366b46e6efee0717a96bd7629bebba0287c1f3986e82142a17f38c5128d57",
        "domain": "numina",
        "question": "The equations of the sides of a quadrilateral are:\n\n$$\ny=-x+7, \\quad y=\\frac{x}{2}+1, \\quad y=-\\frac{3}{2} x+21, \\quad y=\\frac{7}{4} x+\\frac{3}{2}\n$$\n\nDetermine the coordinates of the vertices of the quadrilateral and its area.",
        "answer": "35",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174179,
        "difficulty": null
      },
      {
        "id": "17531d24aa584e748faca16484bfde9ac9b58435bd27b009140b5fa771be9a61",
        "domain": "numina",
        "question": "5. Vova has 19 math grades in his journal, all twos and threes, and the first four grades are twos. It turned out that among the quartets of consecutive grades, all 16 possible combinations of four twos and threes are present. What are Vova's last four grades?",
        "answer": "3222",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 14569,
        "difficulty": null
      },
      {
        "id": "d87e76768707e188cd004fb1afa7648abef01d322a1fb77afb7aadf9153eb0bb",
        "domain": "numina",
        "question": "46th Putnam 1985 Problem A2 ABC is an acute-angled triangle with area 1. A rectangle R has its vertices R i on the sides of the triangle, R 1 and R 2 on BC, R 3 on AC and R 4 on AB. Another rectangle S has it vertices on the sides of the triangle AR 3 R 4 , two on R 3 R 4 and one one each of the other two sides. What is the maximum total area of R and S over all possible choices of triangle and rectangles? Solution",
        "answer": "\\frac{2}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 148772,
        "difficulty": null
      },
      {
        "id": "31e17438529aadc698e87b3855ee7ed601abc9349c0488bf7ac480be325f4bd4",
        "domain": "numina",
        "question": "24. Find the number of 7-digit integers formed by some or all of the five digits, namely, $0,1,2,3$, and 4 , such that these integers contain none of the three blocks 22,33 and 44 .",
        "answer": "29776",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160106,
        "difficulty": null
      },
      {
        "id": "e18863ae5a8010c76939d47600d0b0e7b90029013456e32b30324db7789c415f",
        "domain": "numina",
        "question": "20. A palindrome number is the same either read from left to right or right to left, for example. 121 is a palindrome number. How many 5 -digit palindrome numbers are there together?",
        "answer": "900",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 160012,
        "difficulty": null
      },
      {
        "id": "d26e77a988bea99d74590821a39953b221f58024b82b0993010e2f06df5bf522",
        "domain": "numina",
        "question": "20. Let $A$ be the sum of all non-negative integers $n$ satisfying\n$$\n\\left\\lfloor\\frac{n}{27}\\right\\rfloor=\\left\\lfloor\\frac{n}{28}\\right\\rfloor .\n$$\n\nDetermine $A$.",
        "answer": "95004",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160755,
        "difficulty": null
      },
      {
        "id": "58f8803d7e3e0ff66f30fde6c1f36957bfa5762774e55d83cd7843a52117694c",
        "domain": "numina",
        "question": "What are the last two digits of $7^{9^{9^{9}}}$?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 78808,
        "difficulty": null
      },
      {
        "id": "f30dbb2ffdf1fd8194d7897e0a8046bb10779e8477e77f6253a7dda5ccb2d883",
        "domain": "numina",
        "question": "2. Determine all natural numbers that are 33 times larger than the sum of their digits.",
        "answer": "594",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 189547,
        "difficulty": null
      },
      {
        "id": "6e4faa65113fc9da7079f244a464f3595fdc4d29618bd7bcad440497f5a75a0f",
        "domain": "numina",
        "question": "5.1. (13 points) On the picture, the entrance and exit of the maze are marked with arrows. You can move through it such that on this picture you can only move right, down, or up (you cannot turn around). How many different ways are there to navigate this maze?\n\n---\n\nThe translation maintains the original text's formatting and structure.",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5912,
        "difficulty": null
      },
      {
        "id": "045246a7bad70180b6e14a823d8cef8135310d08491c1f15ea213a6b533884a2",
        "domain": "numina",
        "question": "Subject (3). Consider the triangle $ABC$ with $\\mathrm{m}(\\angle BAC) = 90^{\\circ}$. The bisector of $\\angle ACB$ intersects the segment $(AB)$ at point $E$. If there exists $D \\in (CE)$ such that $\\mathrm{m}(\\angle DAC) = \\mathrm{m}(\\angle BDE) = x^{\\circ}$, then calculate $x$.\n\nThe statement in the problem is unfortunate. Such a point $D$ always exists, namely $D \\equiv I$, the incenter of $\\triangle ABC$, for which $\\angle IAC = \\angle BIE = 45^{\\circ}$. ${ }^{2}$ Otherwise, by continuity considerations, as $D$ traverses $CE$, from $C$ to $E$, the angle $\\angle DAC$ increases from $0^{\\circ}$ to $90^{\\circ}$, and the angle $\\angle BDE$ increases from $\\frac{1}{2} \\angle C$ to $90^{\\circ} - \\frac{1}{2} \\angle C$. This means that the continuous function $f(D) = \\angle BDE - \\angle DAC$ takes the values $f(C) = \\frac{1}{2} \\angle C > 0$ and $f(E) = -\\frac{1}{2} \\angle C < 0$, so there must exist (at least) one point $D$ for which $f(D) = 0$. If more than one exists, and since the problem asks for a (unique) $x$, the value must be 45. Perhaps this observation should lead to the elegant synthetic solution that follows, where it is demonstrated that $D \\equiv I$ is the only such point. [^1]",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 180240,
        "difficulty": null
      },
      {
        "id": "a91b4ec9d79869595d0b21664401897ac1883f6600e81ff258f8ee91466859eb",
        "domain": "numina",
        "question": "2. Calculate the value of the expression $\\sqrt{(\\sqrt{3}-3)^{2}}-(\\sqrt{3}-3)$.",
        "answer": "6-2\\sqrt{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86911,
        "difficulty": null
      },
      {
        "id": "d7d90c7e46907883ed51444d6665f2dfd511bb4fe51d7d516de86449afa02e4b",
        "domain": "numina",
        "question": "7.101. $\\lg 8-\\lg \\sqrt{x+6}=\\lg 16-\\lg (x-2)$",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50864,
        "difficulty": null
      },
      {
        "id": "3d124f480d7dcfd9c8e623dfcbaa258ec4b280d98e04db675da5f0eb8d2b9e3c",
        "domain": "numina",
        "question": "6. In the Cartesian coordinate system $x O y$, the area of the plane region corresponding to the point set $K=\\{(x, y) \\mid(|x|+|3 y|-6)(|3 x|+|y|-6) \\leqslant 0\\}$ is $\\qquad$ .",
        "answer": "24",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 139309,
        "difficulty": null
      },
      {
        "id": "8cde1edaec3729d8501b6f610ab150c88eb051ca8ac830002b3c5b7c9756fadd",
        "domain": "numina",
        "question": "11. Find the value of\n$$\n\\frac{2011^{2}+2111^{2}-2 \\times 2011 \\times 2111}{25}\n$$",
        "answer": "400",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160702,
        "difficulty": null
      },
      {
        "id": "6b3b9dc6bc8142bc81b64ed085d5a184e08988541ca7b44e1ba290b4c808efaf",
        "domain": "numina",
        "question": "Write the digits $1,2,3,4,5,6,7,8,9$ in some order to form a nine-digit number $\\overline{\\operatorname{abcdefghi}}$. If $A=\\overline{a b c}+\\overline{b c d}+\\overline{c d e}+\\overline{d e f}+\\overline{e f g}+\\overline{f g h}+\\overline{g h i}$, find the maximum possible value of $A$.",
        "answer": "4648",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 107934,
        "difficulty": null
      },
      {
        "id": "7c91408124d353f5c6ddc4d87848f422839eaab00526eed2e3766869cd406e0b",
        "domain": "numina",
        "question": "1. Maurits cycles to school every day. He can choose between two different routes. Route B is $1.5 \\mathrm{~km}$ longer than route A, but because he encounters fewer traffic lights along the way, his average speed on route B is $2 \\mathrm{~km} / \\mathrm{u}$ higher than on route A. As a result, his cycling trip via route A takes exactly as long as via route B.\n\nHow long does his cycling trip to school take?",
        "answer": "45",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 85816,
        "difficulty": null
      },
      {
        "id": "c3c66867a30a0110092fc3ded35b8e01fff5b6ae161b1667d9f15c88575021d4",
        "domain": "numina",
        "question": "From a barrel, we pour out 4 liters of wine and replace it with 4 liters of water. From the resulting mixture, we again pour out 4 liters and replace it with 4 liters of water. We repeat this process a total of three times, and in the final result, there is 2.5 liters more water than wine. How many liters of wine were originally in the barrel?",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 174980,
        "difficulty": null
      },
      {
        "id": "d4e98599dda04a8912760ac94148f8dc8aa950726c39ab197308c047f2e3aa96",
        "domain": "numina",
        "question": "3. Find the smallest distance from the point with coordinates $(5 ; 10 ; 13)$ to a point whose coordinates are positive and satisfy the inequality\n\n$$\n(x+y+z)\\left(\\frac{1}{x+y}+\\frac{1}{y+z}+\\frac{1}{z+x}\\right) \\geq 4.5 \\sqrt[4]{1-(2 y+x)^{2}}\n$$\n\nIn your answer, write the square of the found distance.",
        "answer": "115.2",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 10088,
        "difficulty": null
      },
      {
        "id": "f2e7192fd938eaf37f5817fa7a34c99f176de685c1fd722fc7178ac09b3497c6",
        "domain": "numina",
        "question": "Sonkin $M$.\n\nIn trapezoid $A B C D$ with area 1, the bases $B C$ and $A D$ are in the ratio $1: 2$. Let $K$ be the midpoint of diagonal $A C$. Line $D K$ intersects side $A B$ at point $L$. Find the area of quadrilateral $B C K L$.",
        "answer": "\\frac{2}{9}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 65143,
        "difficulty": null
      },
      {
        "id": "53f823b93338acf2a12918092d3a8ad341bd36f259114e6ac0bce22df33ed9d5",
        "domain": "numina",
        "question": "Example 5 Color each vertex of a 2003-gon with one of three colors: red, blue, or green, such that adjacent vertices have different colors. How many such colorings are there? ${ }^{[3]}$\n(2002-2003, Hungarian Mathematical Olympiad)",
        "answer": "2^{2003}-2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 133775,
        "difficulty": null
      },
      {
        "id": "9efbe526a0347c66dfbcc14351a18f27ba02716aa9b16f3512210d80aabca28f",
        "domain": "numina",
        "question": "5.19 It is known that 1982 people attended a gathering, and among any 4 people, at least 1 person knows the other 3. How many people at least know all the attendees at this gathering?",
        "answer": "1979",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 106401,
        "difficulty": null
      },
      {
        "id": "016c90f0634ec4c9b8a7249ad6d8af8955a21343d00795cc92c8794a5b8023c2",
        "domain": "numina",
        "question": "2. The cells of a $100 \\times 100$ table are painted white. In one move, it is allowed to select any 99 cells from one row or one column and repaint each of them to the opposite color - from white to black, and from black to white. What is the minimum number of moves required to obtain a table with a checkerboard pattern of cells?",
        "answer": "100",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18911,
        "difficulty": null
      },
      {
        "id": "8bce438fc6ff0f4eadd537908acdc21c54b2a25a04651fa6c5cc330bca2a57fb",
        "domain": "numina",
        "question": "## [ Product of lengths of chord segments and lengths of secant segments\n\n Pythagorean Theorem (direct and inverse). [Cyclic quadrilateral with perpendicular diagonals] The diagonals $A C$ and $B D$ of the cyclic quadrilateral $A B C D$ are perpendicular to each other and intersect at point $M$. It is known that $A M=3, B M=4$ and $C M=6$. Find $C D$.\n\n#",
        "answer": "7.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66255,
        "difficulty": null
      },
      {
        "id": "b63f167a1d16b9486b96bc657945400a9c3b511d256c000c1a715950975fdd04",
        "domain": "numina",
        "question": "18. There is a sequence, the first number is 6, the second number is 3, starting from the second number, each number is 5 less than the sum of the number before it and the number after it. What is the sum of the first 200 numbers in this sequence, from the first number to the 200th number?",
        "answer": "999",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134990,
        "difficulty": null
      },
      {
        "id": "14d039d576e0f6719c81ae116fc18d061ee2e5578319000765d7a4991be6b9e0",
        "domain": "numina",
        "question": "In the case of Sándor Mátyás, the leader of the patriots sent an equation to his friends:\n\n$$\n2 x^{2}+4 x y+7 y^{2}-12 x-2 y+N=0\n$$\n\nThe secret message was that the uprising should take place on the $N$-th day of the month. The patriots often said that the uprising was the only solution. This also characterized $N$: the equation was satisfied by only one pair of real numbers $(x, y)$. What is the value of $N$?",
        "answer": "23",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171252,
        "difficulty": null
      },
      {
        "id": "252456ffc47043bf692acd04c5b78da8d6208147d3847211fd9c54655fb10328",
        "domain": "numina",
        "question": "H4. The points $A, B$ and $C$ are the centres of three faces of a cuboid that meet at a vertex. The lengths of the sides of the triangle $A B C$ are 4,5 and 6 .\nWhat is the volume of the cuboid?",
        "answer": "90\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 155230,
        "difficulty": null
      },
      {
        "id": "2ed423d7711b8ee0c3485f8ec24bbc595a3ad8ba53279037417e2c9e535dc1f2",
        "domain": "numina",
        "question": "1. Variant 1.\n\nPetya has four cards with digits $1,2,3,4$. Each digit appears exactly once. How many natural numbers greater than 2222 can Petya make from these cards?",
        "answer": "16",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 12554,
        "difficulty": null
      },
      {
        "id": "58f95075d8f1f5aa3c6d58f3fe9d73059a91c39860513cf5a3105f8b290549d7",
        "domain": "numina",
        "question": "4. Points $A, B, C, D$ lie on the circumference of a circle, and $B C=D C=4, A E=6$, the lengths of segments $B E$ and $D E$ are both integers, find the length of $B D$.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 141970,
        "difficulty": null
      },
      {
        "id": "e7c20bd770b33caec9542a2efd9ad100584bc9770700508a2f0628d0a3d37d49",
        "domain": "numina",
        "question": "1. A palindrome is a number that reads the same from left to right and from right to left. For example, the numbers 353 and $4884-$ are palindromes. It is known that a three-digit number $x$ is a palindrome. To it, 32 was added and a four-digit number was obtained, which is also a palindrome. Find $x$.",
        "answer": "969",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 4051,
        "difficulty": null
      },
      {
        "id": "0523d811fb4b7b9d0642b3ed88d38504edab520230a9d9b79e82cc6a76c7476a",
        "domain": "numina",
        "question": "[Example 3.6.6] Find all positive integers $n>1$, such that $\\frac{2^{n}+1}{n^{2}}$ is an integer.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 100351,
        "difficulty": null
      },
      {
        "id": "60a1a4ce6ee0616d5e4596e7bc56967a2e65b4931a9261869507c49e7450b4f6",
        "domain": "numina",
        "question": "## Task B-2.3.\n\nThere are 8 steps to the top of the staircase. In how many ways can we reach the top if we can climb by one step or by two steps?",
        "answer": "34",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89277,
        "difficulty": null
      },
      {
        "id": "4ff4bbac33196bd588e2db6037c1bbe3966af4d1323c36bada62c1fc63158b03",
        "domain": "numina",
        "question": "Problem 3. There were several whole cheese heads on the warehouse. At night, rats came and ate 10 heads, and everyone ate equally. Several rats got stomachaches from overeating. The remaining 7 rats the next night finished off the remaining cheese, but each rat could eat only half as much cheese as the night before. How much cheese was originally on the warehouse? [5 points] (A.D. Blyunkov, I. V. Raskina)",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 22749,
        "difficulty": null
      },
      {
        "id": "021061e753c48f18a3df8233da04098b43824ee49439a157fbb7c10405690516",
        "domain": "numina",
        "question": "5. Calculate: $1+3 \\frac{1}{6}+5 \\frac{1}{12}+7 \\frac{1}{20}+9 \\frac{1}{30}+11 \\frac{1}{42}+13 \\frac{1}{56}+15 \\frac{1}{72}+17 \\frac{1}{90}$",
        "answer": "81\\frac{2}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 106667,
        "difficulty": null
      },
      {
        "id": "a09d9c8488e411270340c1801066dfce229a5ad05f24dd52e1590ddd6b6fe690",
        "domain": "numina",
        "question": "6. Convex quadrilateral $A B C D$ satisfies $\\angle C A B=\\angle A D B=30^{\\circ}, \\angle A B D=77^{\\circ}, B C=C D$, and $\\angle B C D=n^{\\circ}$ for some positive integer $n$. Compute $n$.",
        "answer": "68",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151995,
        "difficulty": null
      },
      {
        "id": "ce7632b9361547639474ca8dc6f177aaefa0768ad54553b83a43eb192a8c2bdb",
        "domain": "numina",
        "question": "Question 182, In a group of 9 people, arrange for each person to shake hands with exactly two other people in the group. Two different handshake arrangements are considered distinct if and only if there are at least two people who shake hands in one arrangement but do not shake hands in the other. How many different handshake arrangements are there? (2012 AIME Problem)",
        "answer": "30016",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 115077,
        "difficulty": null
      },
      {
        "id": "49eea9e3637486103245f11bcd586fabca808fba824806f2fba6fd4b9a47df60",
        "domain": "numina",
        "question": "1. Let the set $A=\\{1,2,3, \\cdots, 99\\}, B=\\{2 x \\mid x \\in A\\}, C=\\{x \\mid 2 x \\in A\\}$, then the number of elements in $B \\cap C$ is $\\qquad$ ـ.",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 101480,
        "difficulty": null
      },
      {
        "id": "24eaa8e707610192d309c8556f4fdec1e4fad2a2397263323ab3ca14a4a20189",
        "domain": "numina",
        "question": "$24, 2$ red balls and $11$ white balls are arranged in a row, satisfying the following conditions: the red balls are not adjacent, and in any consecutive $7$ balls, there is at least one red ball, then there are $\\qquad$ ways to arrange them.",
        "answer": "31",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146226,
        "difficulty": null
      },
      {
        "id": "00a4d8c835a14c0489fa341459271338a1c95b2acdc6f298226f16529b13ce4c",
        "domain": "numina",
        "question": "5. Two circles touch internally at point $A$. From the center of the larger circle $O$, a radius $O B$ of the larger circle is drawn, which is also tangent to the smaller circle at point $C$. Determine $\\measuredangle B A C$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188068,
        "difficulty": null
      },
      {
        "id": "6db5db2958abbdfd749efd289aad3f4e8c0ebbdf1db8253d982fc76b5a97d1b4",
        "domain": "numina",
        "question": "139. Moving Digits. A certain number, which is written using fewer than 30 digits, starts (if we move from left to right) with the digits 15, that is, it has the form $15 \\ldots$. If we multiply it by 5, the result can be obtained simply by moving these two digits to the right end; in the end, we get a number of the form ... 15. Find the original number.",
        "answer": "157894736842105263",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41572,
        "difficulty": null
      },
      {
        "id": "3451618cfea88c65a3636656a13cc28c9d50f8e3e5d283a395f67854317f0938",
        "domain": "numina",
        "question": "2. Place nine small balls numbered $1, 2, \\cdots, 9$ into a bag. These balls are identical except for their numbers. Person A draws a ball from the bag, with the number being $a$, and then returns it. Person B then draws a ball from the bag, with the number being $b$. The probability that the inequality $a - 2b + 10 > 0$ holds is $\\qquad$.",
        "answer": "\\frac{61}{81}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 127720,
        "difficulty": null
      },
      {
        "id": "886deed8cb8aaf81fd7fea9561efbb05ba8f13e063cb4f9ace62a9c0c1695673",
        "domain": "numina",
        "question": "Problem 8.1. A square was cut into four equal rectangles, and from them, a large letter P was formed, as shown in the figure, with a perimeter of 56.\n![](https://cdn.mathpix.com/cropped/2024_05_06_6da73bfd3e09e8b55e3fg-27.jpg?height=356&width=720&top_left_y=274&top_left_x=366)\n\nWhat is the perimeter of the original square",
        "answer": "32",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12889,
        "difficulty": null
      },
      {
        "id": "f3654fe08434b10957307ce6339f9299e017b2d47fc12cb4d4b7df859f33c244",
        "domain": "numina",
        "question": "5. The natural numbers $1,2, \\ldots, 64$ are written in the cells of an $8 \\times 8$ table such that for all $k=1,2,3, \\ldots, 63$ the numbers $k$ and $k+1$ are in adjacent cells. What is the maximum possible value of the sum of the numbers on the main diagonal?",
        "answer": "432",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 4566,
        "difficulty": null
      },
      {
        "id": "5604ef67788e5d5b665ec5e20d06111ae022fa1397a40feff9857a315c4c5008",
        "domain": "numina",
        "question": "2. A positive integer $n$ is said to be 'good' if $n^{2}-1$ can be written as the product of three distinct prime numbers. Find the sum of the five smallest 'good' integers.\n(1 mark)\n對於正整數 $n$, 若 $n^{2}-1$ 可寫成三個不同質數之積, 則 $n$ 稱為「好數」。求最小的五個「好數」之和。",
        "answer": "104",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153641,
        "difficulty": null
      },
      {
        "id": "6ea7626f4da2336124f2fcfc682b9baee7ac1d9827afa6f60f2d659832ebfe06",
        "domain": "numina",
        "question": "Problem 6.3. In a new math textbook, there are a total of 91 problems. Yura started solving them in the mornings, beginning on September 6.\n\nEvery morning, starting from September 7, he solves one fewer problem than the previous morning (until the problems run out).\n\nBy the evening of September 8, Yura realized that there were still 46 problems left to solve in the textbook. On which day of September will he finish solving the textbook?",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12094,
        "difficulty": null
      },
      {
        "id": "da61cda7548fce2d86ac7ad6fa1209a534bd079a3b1c7934272ca324cd18a4a9",
        "domain": "numina",
        "question": "2. (13 points) A triathlon competitor swam 1 km in the first stage. In the second stage, he cycled 25 km, and in the third stage, he ran 4 km. He completed the entire distance in 1 hour 15 minutes. Before the competition, he tried out the course: he swam for $1 / 16$ hours, cycled and ran for $1 / 49$ hours each, covering a total of 5/4 km. During the competition, he completed each stage at the same speed as during the trial run. How long did he cycle and at what speed?",
        "answer": "\\frac{5}{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4235,
        "difficulty": null
      },
      {
        "id": "dc9bc357383058fb18624453ca54d08a30faa34a37b2bcd2968777e57f874447",
        "domain": "numina",
        "question": "João's scientific calculator has a special key that transforms any number $x$ displayed on the screen and that is different from 1 into the number $\\frac{1}{1-x}$.\n\na) What happens if the number 2 is displayed on the screen and we press the special key three times?\n\nb) What happens if the number 2 is displayed on the screen and we press the special key ten times?\n\nc) Finally, what happens if the number 2 is displayed on the screen and we press the special key 2015 times?",
        "answer": "\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73517,
        "difficulty": null
      },
      {
        "id": "a5cfa86374c3c29b685daea6be6efa71433568523509a915caa2b88dd06a896c",
        "domain": "numina",
        "question": "$\\begin{array}{ll}{\\left[\\begin{array}{l}\\text { Game Theory (other) }\\end{array}\\right]} \\\\ {\\left[\\begin{array}{l}\\text { Pairings and Groupings; Bijections }\\end{array}\\right]}\\end{array}$\n\nA token is placed on a chessboard. Two players take turns moving the token to an adjacent square. It is forbidden to place the token on a square it has already visited. The player who cannot make a move loses. Who wins with correct play?",
        "answer": "64",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 54357,
        "difficulty": null
      },
      {
        "id": "de383df833fe9f978bfbe4671a96690939a4c79964c629c4c940f857412a4549",
        "domain": "numina",
        "question": "2. For a natural number ending not in zero, one of its digits was replaced by zero (if it is the leading digit, it was simply erased). As a result, the number decreased by 9 times. How many numbers exist for which this is possible?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 7660,
        "difficulty": null
      },
      {
        "id": "1b87f593c1efaed5396b7eefcabf03a332dd6863113a8b037f057bdf7d380611",
        "domain": "numina",
        "question": "7. What? Where? When? The game wheel in the TV quiz is \n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0312347ad1209bb36d0eg-04.jpg?height=457&width=665&top_left_y=1693&top_left_x=296)\ndivided into 13 identical sectors. The sectors are numbered from 1 to 13. At the beginning of the game, each sector contains an envelope with a question. Players choose a random sector using a spinner with an arrow. If this sector has already been chosen before, there is no envelope in it, and then the next sector clockwise is played. If it is also empty, the next one, and so on, until a non-empty sector is found. Before the break, the players played 6 sectors. a) (from 6th grade, 1 point) What is more likely: that among the played sectors there is sector №1 or that among the played sectors there is sector №8?\n\nb) (from 7th grade, 5 points). Find the probability that as a result, the six sectors played in a row have numbers from №1 to №6.",
        "answer": "0.00348",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10978,
        "difficulty": null
      },
      {
        "id": "8e5f6f32688369ea9a99df83888b09bdbd3d60c8e386fc68dfb285ca6c3b66a7",
        "domain": "numina",
        "question": "58. From the 205 positive integers $1,2,3, \\cdots \\cdots, 205$, the maximum number of integers that can be selected such that for any three selected numbers $a, b, c(a<b<c)$, we have $a b \\neq c$ is $\\qquad$.",
        "answer": "193",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 109441,
        "difficulty": null
      },
      {
        "id": "57f6cdca64607a969a6723a05c77dbd454f857a52e586f5b33537ccf1b89be48",
        "domain": "numina",
        "question": "Calculate\n\n$$\n\\sqrt{32 \\sqrt{16 \\sqrt{8 \\sqrt{4}}}}\n$$\n\nOnly a numerical answer is expected here.",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 80221,
        "difficulty": null
      },
      {
        "id": "150bf25c911adcf0b9a28676b9adc21938621b9d39930b34446d342dc787389e",
        "domain": "numina",
        "question": "We draw the diagonals of the convex quadrilateral $A B C D$, and then the centroids of the 4 triangles thus formed. What fraction of the area of quadrilateral $A B C D$ is the area of the quadrilateral determined by the 4 centroids?",
        "answer": "\\frac{2}{9}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 168110,
        "difficulty": null
      },
      {
        "id": "0ebbd551408f091d25827872a165cb2cb887ced8d0c0588f29d91c3432b422b2",
        "domain": "numina",
        "question": "$3 \\cdot 26$ If real numbers $a, b, x, y$ satisfy $a x+b y=3, a x^{2}+b y^{2}=7, a x^{3}+$ $b y^{3}=16, a x^{4}+b y^{4}=42$, find the value of $a x^{5}+b y^{5}$.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135333,
        "difficulty": null
      },
      {
        "id": "23bc4a511fd263a7e4bbea5c8144421c4a2ae32d09e434650ade10d455964d33",
        "domain": "numina",
        "question": "The positive integer $d$ has the property that each of 468,636, and 867 has the same remainder, $r$, when divided by $d$. What is the largest possible value of $d+r$ ?",
        "answer": "27",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194471,
        "difficulty": null
      },
      {
        "id": "acd9c5ce75578dcd9f4bcb0430e6bc9b6e8d094e5cca05fddaef318d71e1c092",
        "domain": "numina",
        "question": "8 Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=\\sqrt{x^{2}+1}+\\sqrt{(x-12)^{2}+16}$ is $\\qquad$",
        "answer": "13",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110492,
        "difficulty": null
      },
      {
        "id": "2311ab02b21c5aefb3ec8d5b31dfedc937eda1ab18d646b6c15319690987d925",
        "domain": "numina",
        "question": "It is known that in the small lottery, 5 numbers are drawn from 90. If someone bets on 1, 2, 3, 4, or 5 numbers, what is the probability that these numbers will be drawn? In the past, when the small lottery was still in operation, the state paid out 14, 240, 4800, 60000, and 100000 times the stake for estratra (one number), ambora (two numbers), ternora, quaternora, and quinternora, respectively.\n\nWhat percentage of all stakes was the state's revenue?",
        "answer": "99.78",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 172110,
        "difficulty": null
      },
      {
        "id": "bc29b0f0a2e9fb4c098ec0bd6a69fc2031f1a73b91946d47c7ed167efd406bba",
        "domain": "numina",
        "question": "11. (12 points) Person A and Person B start from point A to point B at the same time: A is faster than B. After A reaches point B, A's speed doubles and A immediately returns to point A, meeting B 240 meters away from point B; after B meets A, B's speed also doubles, and B turns around to return; when A returns to point A, B is still 120 meters away from point A. What is the distance between points A and B in meters?",
        "answer": "420",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 144224,
        "difficulty": null
      },
      {
        "id": "24a57c5318e3a175a2962cdde0f123db349c264feb7919d9851f398739009849",
        "domain": "numina",
        "question": "B1. At least how many times must we roll two fair dice simultaneously so that the probability of rolling the same number of pips on both dice at least once is greater than $\\frac{1}{2}$?\n\n(20 points)",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 163148,
        "difficulty": null
      },
      {
        "id": "b91205d8ec182bc249ac5d2f8559f8deb807f83fcaff9bd5bb936d33c720d337",
        "domain": "numina",
        "question": "Example 4. The mathematical expectations of two independent random variables $X$ and $Y$ are known: $M(X)=4, M(Y)=5$.\n\nFind the mathematical expectation of their product.",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 33011,
        "difficulty": null
      },
      {
        "id": "a0ac57c686d2ee74741cfdb51769c207d31d33694c3d6edf91db4efbe5552bf9",
        "domain": "numina",
        "question": "Three friends Pankrác, Servác, and Bonifác went on a night walk through a natural labyrinth during their vacation. At the entrance, each received a candle and they set off in different directions. All of them successfully navigated the labyrinth, but each took a different path. In the following square grid, their paths are marked. We know that Pankrác never went south and that Servác never went west. How many meters did Bonifác walk in the labyrinth if Pankrác walked exactly $500 \\mathrm{~m}$?\n\n(M. Petrová)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_0d982c5cbb35b07eed65g-1.jpg?height=665&width=1114&top_left_y=787&top_left_x=471)\n\nIdea. Which path did Bonifác definitely not take?",
        "answer": "625",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 76603,
        "difficulty": null
      },
      {
        "id": "fe75ea8f22739f2dfe22c1ecacf6b9e7097e818609963ac4db9794a65e741479",
        "domain": "numina",
        "question": "10.1. Find the sum $\\sin x + \\sin y + \\sin z$, given that $\\sin x = \\tan y$, $\\sin y = \\tan z$, $\\sin z = \\tan x$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14432,
        "difficulty": null
      },
      {
        "id": "3c7714397712c0a798712f551cc8424b25f7949eabca9def1e67c88f046a168a",
        "domain": "numina",
        "question": "1. Determine the geometric locus of points $M$ in the Cartesian coordinate plane such that the tangents drawn from point $M$ to the parabola $y=x^{2}$ are mutually perpendicular.",
        "answer": "-\\frac{1}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88174,
        "difficulty": null
      },
      {
        "id": "2dfb5a657c9c99d64504075b0c4b6ef0f934d75196bdaf48e574712169ecd0e1",
        "domain": "numina",
        "question": "43. As shown in the figure, two satellites $\\mathrm{A}, \\mathrm{B}$ are both orbiting the Earth's center in a counterclockwise direction at constant speeds. It is known that the time ratio for $\\mathrm{A}$ and $\\mathrm{B}$ to complete one orbit is 1:5. Starting from the positions shown in the figure, during the time it takes for $\\mathrm{B}$ to complete one orbit, satellites $\\mathrm{A}$ and $\\mathrm{B}$ and the Earth's center are collinear $\\qquad$ times.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 137761,
        "difficulty": null
      },
      {
        "id": "011aa06bfce76b02ad5d0c5233ef6a2d62a7c83ddb632bb2a9278c9b205a3434",
        "domain": "numina",
        "question": "1.047. $\\left(\\frac{\\sqrt{561^{2}-459^{2}}}{4 \\frac{2}{7} \\cdot 0.15+4 \\frac{2}{7}: \\frac{20}{3}}+4 \\sqrt{10}\\right): \\frac{1}{3} \\sqrt{40}$.",
        "answer": "125",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50370,
        "difficulty": null
      },
      {
        "id": "1a1e6af7738796f0151d7328fd12f3cb3479c3ada5a5bfcf19422a3ae3ebc8fb",
        "domain": "numina",
        "question": "12. Find the remainder of the division of $67^{283}$ by 13.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 34743,
        "difficulty": null
      },
      {
        "id": "94d31dcb19a0ec11b8be43aacb7142e93be573a0967d5d0b8966a0cebbd8c21a",
        "domain": "numina",
        "question": "5. The maximum value of the orthogonal projection area of a regular tetrahedron with edge length 1 on a horizontal plane is $\\qquad$",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 143942,
        "difficulty": null
      },
      {
        "id": "fb7f7666a4a1f61dfc0d5508234c645e7f418865702cffc2ee1ad1ff8783aebc",
        "domain": "numina",
        "question": "8. Suppose when repeatedly tossing a fair coin, the probability of getting five consecutive heads before two consecutive tails is $p$. Then the value of $p$ is $\\qquad$ .",
        "answer": "\\frac{3}{34}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 92018,
        "difficulty": null
      },
      {
        "id": "59dd935791898b29869f8ac4d9de2f76081162f7eb873d07d825991a01ca7835",
        "domain": "numina",
        "question": "7. There are two strips of paper, one 21 cm long and the other 13 cm long. After cutting off the same length from both strips, it is found that the remaining length of the shorter strip is thirteen-eighths of the remaining length of the longer strip. How long was the piece that was cut off?",
        "answer": "0.2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107657,
        "difficulty": null
      },
      {
        "id": "bde2813aaffd2b6a7cbc5f5b63e771e7f2f6e78ea7c9cd0b14d81605bf401079",
        "domain": "numina",
        "question": "10. The recurring decimal $0 . \\dot{x} y \\dot{z}$, where $x, y, z$ denote digits between 0 and 9 inclusive, is converted to a fraction in lowest term. How many different possible values may the numerator take?\n(1 mark)\n若把循環小數 $0 \\dot{x} y \\dot{z}$ （其中 $x 、 y 、 z$ 代表 0 至 9 之間的數字，包括 0 和 9）化戌最簡分數，分子有多少個不同的可能値？",
        "answer": "660",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153580,
        "difficulty": null
      },
      {
        "id": "1e2ba8c36f6e4bdee8c1ddbbdb23e6d42060831edc68669d3eddef25c8e50269",
        "domain": "numina",
        "question": "A 100-row and 100-column table has all 1s in its first column, and in the $k$-th row, there is an arithmetic sequence with a difference of $k$. Among the numbers located along the diagonal stretching from the bottom left to the top right of the table, which is the largest?",
        "answer": "2501",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170083,
        "difficulty": null
      },
      {
        "id": "6b75f0d63f895873e7a43648bcc44de392e7236a3964b1e2a352926333fdc625",
        "domain": "numina",
        "question": "## Problem B2\n\n$\\mathrm{P}_{1}, \\mathrm{P}_{2}, \\ldots, \\mathrm{P}_{\\mathrm{n}}$ are points in the plane and $\\mathrm{r}_{1}, \\mathrm{r}_{2}, \\ldots, \\mathrm{r}_{\\mathrm{n}}$ are real numbers such that the distance between $P_{i}$ and $P_{j}$ is $r_{i}+r_{j}$ (for $i$ not equal to $j$ ). Find the largest $n$ for which this is possible.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 187313,
        "difficulty": null
      },
      {
        "id": "f9c401d0484756754c0c0612c7541b5a9781953d2fa4bb75bf29d4c62f2a3fed",
        "domain": "numina",
        "question": "Let's calculate as simply as possible the value of\n\n$$\na^{3}+b^{3}+3\\left(a^{3} b+a b^{3}\\right)+6\\left(a^{3} b^{2}+a^{2} b^{3}\\right)\n$$\n\nif $a+b=1$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 173456,
        "difficulty": null
      },
      {
        "id": "c5e5a49c6936f8560f1602323560e9fa959d7b925012df0d66f241a281bc9344",
        "domain": "numina",
        "question": "3. Find all natural numbers $n$ for which $2^{n}+n^{2016}$ is a prime number.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 11834,
        "difficulty": null
      },
      {
        "id": "157beb0f637fca510d73047dee5d762b847b67daf1a854b25bced8f29d5cb7ee",
        "domain": "numina",
        "question": "Task B-3.4. In the plane, 100 concentric circles are drawn with radii $1 \\mathrm{~cm}, 2 \\mathrm{~cm}, 3 \\mathrm{~cm}, \\ldots, 100 \\mathrm{~cm}$. The circle with a radius of $1 \\mathrm{~cm}$ is painted red, and all other areas bounded by consecutive circles are painted either red or green, but in such a way that adjacent areas are painted different colors. Determine the ratio of the area of all green regions to the area of all red regions.",
        "answer": "\\frac{101}{99}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88210,
        "difficulty": null
      },
      {
        "id": "c862fc5ee7433306817519b623106e701d9139e4db7a2625ed78b3f26b08f03a",
        "domain": "numina",
        "question": "For each positive integer $n$, the expression $1+2+3+\\cdots+(n-1)+n$ represents the sum of all of the integers from 1 to $n$ inclusive. What integer is equal to\n\n$$\n(1+2+3+\\cdots+2020+2021)-(1+2+3+\\cdots+2018+2019) ?\n$$",
        "answer": "4041",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194227,
        "difficulty": null
      },
      {
        "id": "521828c56ec3d3b815f2cd2fd10799cfdffc62af1d528427baffb4147a7fd144",
        "domain": "numina",
        "question": "2. (1985 National High School League Question) A football invitational tournament involves sixteen cities, each sending two teams, Team A and Team B. According to the competition rules, each pair of teams plays at most one match, and teams from the same city do not play against each other. After several days of the tournament, a statistical review found that, except for Team A from City A, all other teams have played a different number of matches. How many matches has Team B from City A played? Please prove your conclusion.",
        "answer": "15",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 119425,
        "difficulty": null
      },
      {
        "id": "b9cf18f33d81241a8ac5964f498ac3adee9aa57d68f194e2b64fa44b3dda956a",
        "domain": "numina",
        "question": "Four points lying on one circle Auxiliary circle\n\nIn a convex quadrilateral $A B C D$, it is known that $\\angle B C D=80^{\\circ}, \\angle A C B=50^{\\circ}$ and $\\angle A B D=30^{\\circ}$. Find the angle $A D B$.",
        "answer": "50",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64779,
        "difficulty": null
      },
      {
        "id": "a72a36614792863c167e614281126d90b21243c5f1c7a94c9fd52e9283af8034",
        "domain": "numina",
        "question": "2. In the laboratory, there are flasks of two sizes (volume $V$ and volume $V / 2$) in a total of 100 pieces, with at least three flasks of each size. The lab assistant randomly selects three flasks in sequence, and fills the first one with an 80% salt solution, the second one with a 50% salt solution, and the third one with a 20% salt solution. Then he pours the contents of these three flasks into one dish and determines the percentage of salt in it. For what minimum number of large flasks $N$ will the event \"the percentage of salt in the dish is between $45\\%$ and $55\\%$ inclusive\" occur less frequently than the event \"when two fair coins are tossed, one head and one tail appear (in any order)\"? Justify your answer. (16 points)",
        "answer": "46",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10602,
        "difficulty": null
      },
      {
        "id": "05967b86aab774746a3105e1b8b826cbcce64290f267501d526b1b1dfd6e33a6",
        "domain": "numina",
        "question": "[ Limitedness, monotonicity]\n\nGiven a strictly increasing function \\$f \\colon \\mathbb{\\mathrm{N}}_0 \\to \\mathbb{\\mathrm{N}}_0\\$ (where $\\mathbb{\\mathrm{N}}_0$ is the set of non-negative integers), which satisfies the relation $\\mathrm{f}(\\mathrm{n}+\\mathrm{f}(\\mathrm{m}))=\\mathrm{f}(\\mathrm{n})+\\mathrm{m}+1$ for any $m,n \\in \\mathbb{\\mathrm{N}}_0$. Find all possible values that \\$f(2023)\\$ can take.",
        "answer": "2024",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 67810,
        "difficulty": null
      },
      {
        "id": "e47430cf6cf90302a67c498054515970f97b1e32e2b498d5b83dbfa595d1db2c",
        "domain": "numina",
        "question": "F5 (20-3, UK) Let $f, g: \\mathbf{N}^{*} \\rightarrow \\mathbf{N}^{*}$ be strictly increasing functions, and\n$$\nf\\left(\\mathbf{N}^{*}\\right) \\cup g\\left(\\mathbf{N}^{*}\\right)=\\mathbf{N}^{*}, f\\left(\\mathbf{N}^{*}\\right) \\cap g\\left(\\mathbf{N}^{*}\\right)=\\varnothing, g(n)=f(f(n))+1 .\n$$\n\nFind $f(240)$.",
        "answer": "388",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118643,
        "difficulty": null
      },
      {
        "id": "5b8308d71118ed5e190ce070c347118bbc7df78a03794212efee521c57909c73",
        "domain": "numina",
        "question": "10. (20 points) Find the smallest value of the parameter $c$ such that the system of equations has a unique solution\n\n$$\n\\left\\{\\begin{array}{l}\n8(x+7)^{4}+(y-4)^{4}=c \\\\\n(x+4)^{4}+8(y-7)^{4}=c\n\\end{array}\\right.\n$$",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19483,
        "difficulty": null
      },
      {
        "id": "72d8a8be51880ba12afea3bbd9457d7864bc8f6ab069cb4365cdc5796e9a13e5",
        "domain": "numina",
        "question": "5. The subset $X$ of the set $\\{00,01, \\cdots, 98,99\\}$ satisfies: in any infinite sequence of digits, there are two adjacent digits that form an element of $X$. What is the minimum number of elements that $X$ should contain?\n（52nd Moscow Olympiad problem）",
        "answer": "55",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 123250,
        "difficulty": null
      },
      {
        "id": "85878213e8ee399ebd06f9b116e8a4e997eec1171ff7773f32f0b7a9b0f28415",
        "domain": "numina",
        "question": "1.1. Find the maximum value of the function $\\sin ^{13} x+\\cos ^{14} x$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 161854,
        "difficulty": null
      },
      {
        "id": "9ae7abcd30436919c170ac5c5fbe2ba8888301dff860258982a48f02503faa64",
        "domain": "numina",
        "question": "15. If $a=1.69, b=1.73$ and $c=0.48$, find the value of\n$$\n\\frac{1}{a^{2}-a c-a b+b c}+\\frac{2}{b^{2}-a b-b c+a c}+\\frac{1}{c^{2}-a c-b c+a b}\n$$",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160882,
        "difficulty": null
      },
      {
        "id": "ff5a07ff0a64ca32fab2027b0c185df5df63cd90f19bfeff4a8c4f07837db988",
        "domain": "numina",
        "question": "3. Let $g(x)=\\frac{2}{x^{2}-8x+17}$. Find all possible values of the parameter $a$ such that the inequality holds for all real $x$. In your answer, write the difference between the largest and smallest possible values of the parameter $a$.\n\n(6 points)",
        "answer": "2.1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 10154,
        "difficulty": null
      },
      {
        "id": "a6f9c0e7e536d116199a0d004fe241a705db59be2091e2ee058221eb1821c39a",
        "domain": "numina",
        "question": "## Task 4 - 250514\n\nIn each of the figures a), b), c), the numbers 1, 2, 3, 4, 5, 6, 7, 8, 9 are to be entered into the circles.\n\nEach of these numbers should (in each such entry) occur exactly once. For some circles, the number to be entered is already specified. Furthermore, for each entry, the following should apply:\n\nIf you add the four numbers on each side of the triangle, the sum should be the same for each of the three sides.\n\na) Find an entry in Figure a) where the sum for each of the three sides is 17!\n\nb) Find as many entries as possible in Figure b) where the sum for each of the three sides is 21!\n\nc) Find as many entries as possible in Figure c) where the sum for each of the three sides is the same! For each of these entries, give this value!\n\na)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=314&width=369&top_left_y=865&top_left_x=455)\nb)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=320&width=357&top_left_y=862&top_left_x=878)\nc)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=323&width=351&top_left_y=861&top_left_x=1292)",
        "answer": "20",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 183089,
        "difficulty": null
      },
      {
        "id": "35b226e0d12c40502fa760630277b867d982eb634028c6dd3abfe895a93dcb72",
        "domain": "numina",
        "question": "## SUBJECT III\n\nLet $O$ and $O^{\\prime}$ be the centers of the faces $A B C D$ and $B C C B^{\\prime}$ of the cube $A B C D A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}, A B=a$.\n\na) Find the measure of the angle between the lines $D^{\\prime} O$ and $A^{\\prime} O^{\\prime}$.\n\nb) Calculate the distance from the point $O^{\\prime}$ to the plane ($D^{\\prime} A C$).\n\n(prof. Bunu Iulian, Liceul de Arte Baia Mare)",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 180120,
        "difficulty": null
      },
      {
        "id": "6d15c911fea07b0d96e7d200ce3dbb73c3d83df8c49fb450789bacfa97cf860f",
        "domain": "numina",
        "question": "1. How many tiles with dimensions $15 \\times 15$ cm are needed to tile a wall that is 3 m 6 dm long and 27 dm wide $?$",
        "answer": "432",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 42699,
        "difficulty": null
      },
      {
        "id": "a242e054ca56cde9dfb5550afba1105dc3b36dfdb3ae28e58487d93b11f98eaf",
        "domain": "numina",
        "question": "# Problem 5.\n\nNumbers $a_{1}, b_{1}, a_{2}, b_{2}, \\ldots, a_{n}, b_{n}$ satisfy the conditions\n\n$$\n\\begin{gathered}\na_{1}^{2}+a_{2}^{2}+\\cdots+a_{n}^{2}=2018^{2}, \\quad b_{1}^{2}+b_{2}^{2}+\\cdots+b_{n}^{2}=2017^{2} \\\\\na_{1} b_{1}+a_{2} b_{2}+\\cdots+a_{n} b_{n}=2017 \\cdot 2018\n\\end{gathered}\n$$\n\nFind the ratios $a_{1} / b_{1}, a_{2} / b_{2}, \\ldots, a_{n} / b_{n}$.",
        "answer": "\\frac{2018}{2017}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19976,
        "difficulty": null
      },
      {
        "id": "9a6cf721692255baa524699f7a025e00f4cfda9ef8ee349ce31e67ce60c65872",
        "domain": "numina",
        "question": "4. What is the greatest length that a closed, non-self-intersecting broken line can have, which runs along the grid lines of a $6 \\times 10$ cell field?",
        "answer": "76",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 4101,
        "difficulty": null
      },
      {
        "id": "c0be5e31f59072321aeabb553c81c981061db783f65196219464607f7d58b4b4",
        "domain": "numina",
        "question": "3-5. There is a piece of chain consisting of 60 links, each weighing 1 g. What is the smallest number of links that need to be unbuckled so that from the resulting parts, all weights of 1 g, 2 g, 3 g, \\(\\ldots, 60\\) g can be formed (an unbuckled link also weighs 1 g)",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 26319,
        "difficulty": null
      },
      {
        "id": "be1900a0e6cd85e2de879e995cf2a18ea0186ce92596b8076b12bc5ce411db0d",
        "domain": "numina",
        "question": "A Tim number is a five-digit positive integer with the property that it is a multiple of 15 , its hundreds digit is 3, and its tens digit is equal to the sum of its first (leftmost) three digits. How many Tim numbers are there?",
        "answer": "16",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 194449,
        "difficulty": null
      },
      {
        "id": "3aafe0781b9f3431bdc1c484e292f173eb782ee6b9d7ef6789c30bf63bf433af",
        "domain": "numina",
        "question": "3. Nils has a goose farm. Nils calculated that if he sells 75 geese, the feed will last 20 days longer than if he doesn't sell any. If he buys an additional 100 geese, the feed will run out 15 days earlier than if he doesn't make such a purchase. How many geese does Nils have?",
        "answer": "300",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3955,
        "difficulty": null
      },
      {
        "id": "0d12f241c50f1c1db56f391ed17da20ae5ca4740cebfdc727a28912ddacef15f",
        "domain": "numina",
        "question": "14.24. How many different integer solutions does the inequality $|x|+|y|<100$ have?",
        "answer": "338350",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 38532,
        "difficulty": null
      },
      {
        "id": "edf6e644166f777c054f3ce276c3fe8b3f291f0c21a8fc374d4c13f4e3df534a",
        "domain": "numina",
        "question": "7. From the set $\\{1,2, \\cdots, 100\\}$, randomly select elements $m$ and $n$ (which can be the same). Then the probability that the unit digit of $2^{m}+3^{n}$ is 3 is $\\qquad$ .",
        "answer": "\\frac{3}{16}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 134183,
        "difficulty": null
      },
      {
        "id": "72e25937d8343b82c0382d0e343c84a5853f53d03f7718aaaac9d14cc0f6b41d",
        "domain": "numina",
        "question": "7. 14 A fair coin is tossed ten times, let $\\frac{i}{j}$ be the probability that heads do not appear consecutively, where $i$ and $j$ are coprime, find $i+j$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "73",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 96924,
        "difficulty": null
      },
      {
        "id": "2943b97510568671c41813e12b1b21c49bb37ad5b33a47782a7038da9a4f48dd",
        "domain": "numina",
        "question": "Example 3 Find all pairs $(x, y)$ that satisfy the system of equations $\\left\\{\\begin{array}{l}\\frac{\\cos x}{\\cos y}=2 \\cos ^{2} y \\\\ \\frac{\\sin x}{\\sin y}=2 \\sin ^{2} y\\end{array}\\right.$, where $x$, $y \\in\\left(0, \\frac{\\pi}{2}\\right)$.",
        "answer": "\\frac{\\pi}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140710,
        "difficulty": null
      },
      {
        "id": "1fdb6952cd7d33477171c2d4153851adf05b21d62e9494899892894cf754ca64",
        "domain": "numina",
        "question": "3. For two numbers $a, b$, define a new operation: $a * b=a \\times b+a+b$, then $1 * 2+2 * 3=$",
        "answer": "16",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139202,
        "difficulty": null
      },
      {
        "id": "b29a99f3bbf7ad454e4f39ea01a08eaf4b6146dde4ced550016d093184fc63a4",
        "domain": "numina",
        "question": "If $3^{n}=2$ then what is the value of $27^{2 n}$?",
        "answer": "64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 73080,
        "difficulty": null
      },
      {
        "id": "1801f5af051cf2b5e64e71ac3410eca62a2b9841a2c8347b61a416fd74806298",
        "domain": "numina",
        "question": "1. A game participant was offered 30 questions. For each correct answer, they were awarded 7 points, and for an incorrect answer (or no answer), 12 points were deducted. How many correct answers did the participant give if they scored 77 points?",
        "answer": "23",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39610,
        "difficulty": null
      },
      {
        "id": "f0f8c46d51073366b02b952dcc6ea1c6874005850c3faa4ff82d89d98c737aa2",
        "domain": "numina",
        "question": "1. Given $a, b, c$ are the side lengths of $\\triangle ABC$, satisfying $a^{2}+b^{2}=2018 c^{2}$. Then the value of $(\\cot A+\\cot B) \\tan C$ is $\\qquad$",
        "answer": "\\frac{2}{2017}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 103080,
        "difficulty": null
      },
      {
        "id": "f539bed78bbcfa013ec98073e5a7943ef3c0854945a0d4bbc5c7d6a2565b496d",
        "domain": "numina",
        "question": "2. What is the smallest area that a figure on the plane $Oxy$ can have, located between the lines $x=-26$ and $x=2$, bounded below by the $x$-axis, and above by the tangent to the graph of the function $y=5+\\sqrt{4-x}$ at the point of tangency with abscissa $x_{0}$, where $x_{0}$ lies in the interval $-26 \\leq x_{0} \\leq 2$?\n\n(12 points)",
        "answer": "504",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 21736,
        "difficulty": null
      },
      {
        "id": "f15e25486a288577b61f28852dd22c473c327a9e6fb1f1d8d312355618813f5e",
        "domain": "numina",
        "question": "A bus, a train, and a plane depart from city A to city B at the same time. If I take the bus, which has an average speed of 100 $\\mathrm{km} / \\mathrm{h}$, I will arrive in city B at 8 PM. If I take the train, which has an average speed of $300 \\mathrm{~km} / \\mathrm{h}$, I will arrive in city B at 2 PM. What will be the arrival time of the plane if its average speed is $900 \\mathrm{~km} / \\mathrm{h}$?",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 74666,
        "difficulty": null
      },
      {
        "id": "b38c2c574c9d10609cd5dc6c72e4ac5510874c4d13470649198ff66cedc2c6d7",
        "domain": "numina",
        "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-12.5,12.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
        "answer": "650",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 19709,
        "difficulty": null
      },
      {
        "id": "093f41b6962b746ce9152bf4596a31f2af013d73fddac734cc30dfff3198b0ad",
        "domain": "numina",
        "question": "9.1. (12 points) Two circles touch each other externally at point $K$. On their common internal tangent, point $P$ is marked such that $K P=14$. Through point $P$, two secants are drawn to the circles, with one of them intercepting a chord $A B=45$ on the first circle, and the other intercepting a chord $C D=21$ on the second circle, with point $A$ lying between points $B$ and $P$, and point $C$ lying between points $D$ and $P$. Find the ratio $B C: A D$.",
        "answer": "1.75",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6151,
        "difficulty": null
      },
      {
        "id": "5961bccd88f64a1411cf2b31a9da299f11cb50edd33034a8184402d4f0376327",
        "domain": "numina",
        "question": "16. (3 points) Wang Lei and her sister walk from home to the gym to play badminton. It is known that her sister walks 20 meters more per minute than Wang Lei. After 25 minutes, her sister arrives at the gym. At this point, her sister realizes she forgot the racket and immediately returns home along the same route to get it. She meets Wang Lei 300 meters away from the gym. What is the distance from Wang Lei's home to the gym in meters? $\\qquad$ meters.",
        "answer": "1500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94155,
        "difficulty": null
      },
      {
        "id": "a7fe71f418a9012fbfb56a086a6c0e7ca02979aa81dabfd30088a5fb58245b0a",
        "domain": "numina",
        "question": "1. Given non-zero real numbers $x, y$ satisfy\n$$\n\\begin{array}{l}\n(5 x+y)^{2019}+x^{2019}+30 x+5 y=0 . \\\\\n\\text { Then } \\frac{x}{y}=\n\\end{array}\n$$",
        "answer": "-\\frac{1}{6}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 117268,
        "difficulty": null
      },
      {
        "id": "5d801d029fdaf1a70e9e55cd795849e83d8f69ad81615f73eca2c2f80e67b6a0",
        "domain": "numina",
        "question": "2. In quadrilateral $ABCD$, the diagonals intersect at point $P$, diagonal $AC$ is divided into four equal segments by points $P, N$, and $M$ $(|AP|=|PN|=|NM|=|MC|)$, and diagonal $BD$ is divided into four equal segments by points $L, K$, and $P$ $(|BL|=|LK|=|KP|=|PD|)$. Determine the ratio of the area of quadrilateral KLMN to the area of quadrilateral ABCD.",
        "answer": "\\frac{3}{16}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 75464,
        "difficulty": null
      },
      {
        "id": "afbbec3f9b00c0e6db575f80159cbd4f1b8d1138c82397894adf9ae72fab7288",
        "domain": "numina",
        "question": "Example 22 (Mathematical Problem 1486 from \"Mathematical Bulletin\") Find the maximum value of the function $f(x)=\\frac{x(1-x)}{(x+1)(x+2)(2x+1)}$, $x \\in (0,1]$.",
        "answer": "\\frac{1}{3}(8\\sqrt{2}-5\\sqrt{5})",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 140901,
        "difficulty": null
      },
      {
        "id": "cb955c72905a2754bc9a0845baed9efd6c3345f95e2420a7ab07bed1e2440534",
        "domain": "numina",
        "question": "# Task 3. (8 points)\n\nIn the Sidorov family, there are 3 people: the father works as a programmer with an hourly rate of 1500 rubles. The mother works as a hairdresser at home and charges 1200 rubles per haircut, which takes her 1.5 hours. The son tutors in mathematics and earns 450 rubles per academic hour (45 minutes). Every day, the family needs to prepare food, walk the dog, and go to the store. The mother spends 2 hours preparing food, the father 1 hour, and the son 4 hours. Walking the dog takes 1 hour for any family member. A trip to the store takes 1 hour for the father, 2 hours for the mother, and 3 hours for the son. In addition, each family member sleeps for 8 hours and spends another 8 hours on rest and personal matters.\n\nWhat is the maximum amount of money the family can earn in one day?",
        "answer": "19600",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 11492,
        "difficulty": null
      },
      {
        "id": "2066d6ad97ff758c40d60db4ebead1aa14043298120f7a88df2ac89f28965a36",
        "domain": "numina",
        "question": "If $n$ is a natural number, we denote by $n!$ the product of all integers from 1 to $n$. For example: $5! = 1 \\times 2 \\times 3 \\times 4 \\times 5$ and 13! $=$ $1 \\times 2 \\times 3 \\times 4 \\times 5 \\times \\ldots \\times 12 \\times 13$. By convention, $0! = 1$. Find three different integers $a, b$, and $c$, between 0 and 9 such that the three-digit number $abc$ is equal to $a! + b! + c!$.",
        "answer": "145",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 73623,
        "difficulty": null
      },
      {
        "id": "e61ace9a7c1ab0a1a10608f99bb9e0a472c29998c46562a4ad389876ea953c0a",
        "domain": "numina",
        "question": "G1.3 Let $x, y$ be non-zero real numbers. If $x$ is $250 \\%$ of $y$ and $2 y$ is $c \\%$ of $x$, find the value of $c$.",
        "answer": "80",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152858,
        "difficulty": null
      },
      {
        "id": "a6f884eb39ea6d0c6c3d60e8494a07cd8832dc8f64fe2a71394a2e8295842024",
        "domain": "numina",
        "question": "Let $P$ be an arbitrary interior point of the equilateral triangle $ABC$. Let the feet of the perpendiculars from $P$ to the sides $BC$, $CA$, and $AB$ be $D$, $E$, and $F$ respectively. Determine the value of the ratio\n\n$$\n\\frac{P D+P E+P F}{B D+C E+A F}\n$$",
        "answer": "\\frac{\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 167636,
        "difficulty": null
      },
      {
        "id": "5d4520994ee7bf48aa25dd6964e48a5a7c828e9600c6357b4d8339d21acc2db5",
        "domain": "numina",
        "question": "2.1. The numbers 7, 8, 9, 10, 11 are arranged in a row in some order. It turned out that the sum of the first three of them is 26, and the sum of the last three is 30. Determine the number standing in the middle.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6202285ed0b3c14598f3g-1.jpg?height=254&width=765&top_left_y=1169&top_left_x=154)",
        "answer": "11",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13473,
        "difficulty": null
      },
      {
        "id": "09c84e20a4ccdad4c03cc4ad1f32de9600287e04ca118f643057a9667b321cf7",
        "domain": "numina",
        "question": "For example, let the function $f(x) = ax^2 + 8x + 3$ where $a < 0$. For a given negative number $a$, there is a maximum positive number $l(a)$ such that the inequality $|f(x)| \\leq 5$ holds for the entire interval $[0, l(a)]$. Question: For what value of $a$ is $l(a)$ maximized? Find this maximum $l(a)$ and prove your conclusion.\n(1998 National High School Mathematics Competition Problem)",
        "answer": "\\frac{\\sqrt{5}+1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134113,
        "difficulty": null
      },
      {
        "id": "ee3a2262a48fff0164e1c0f524efefa553e5b47e1800b3c02b62c3877b3c7d77",
        "domain": "numina",
        "question": "6. If in the expression $9 \\times 8 \\times 7 \\times 6 \\times 5 \\times 4 \\times 3 \\times 2 \\times 1$ some of the “ $\\times$ ” are changed to “ $\\div$ ” so that the final result is still a natural number, denoted as $N$, then the smallest $N$ is $\\qquad$ .",
        "answer": "70",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 123695,
        "difficulty": null
      },
      {
        "id": "29768ee3f069a44b886a03a53b4907fb77818afab98a9647e5267a641fde4bb1",
        "domain": "numina",
        "question": "1. Given the set $S=\\{1,2,3, \\cdots, 2000,2001\\}$, where a subset $T$ has the property that for any three elements $x, y, z$ in $T$, $x+y \\neq z$, what is the maximum number of elements in $T$? Maximum value",
        "answer": "1001",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 137602,
        "difficulty": null
      },
      {
        "id": "3709d47f14d368f869daad96289ace14956cfd80286cfdd69fbd24f219f1d735",
        "domain": "numina",
        "question": "10. Given a regular 2017-gon inscribed in a unit circle, let $S$ be the set of distances between any two distinct vertices of the 2017-gon, and $Q=\\prod_{s \\in S} s$. Then $Q^{2}=$ $\\qquad$",
        "answer": "2017",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 100145,
        "difficulty": null
      },
      {
        "id": "68145b2c7fe410473470af6ed323e7361517a3a20a617e40ca7c7dc8f5fb7403",
        "domain": "numina",
        "question": "6. A circle with center $O_{1}$ and radius 2, inscribed in triangle $A B C$, touches side $B C$ at point $D$. A second circle with center $O_{2}$ and radius 4 touches the extensions of sides $A B$ and $A C$, as well as side $B C$ at point $E$. Find the area of quadrilateral $O_{1} D O_{2} E$, if the measure of angle $A C B$ is $120^{\\circ}$. (20 points)",
        "answer": "\\frac{30}{\\sqrt{3}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 10195,
        "difficulty": null
      },
      {
        "id": "4320a9363e695604995ec3027d81c93ff9ce56541b02dc5d075240b01a875323",
        "domain": "numina",
        "question": "64 The base of the triangular pyramid $S-ABC$ is an equilateral triangle $ABC$, with a side length of 4. It is also known that $AS=BS=\\sqrt{19}$, and $CS=3$. Find the surface area of the circumscribed sphere of this triangular pyramid.",
        "answer": "4\\pi\\sqrt{\\frac{61}{11}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 134992,
        "difficulty": null
      },
      {
        "id": "50588ee11fa366c2737ef975fdb3038a479fc822a52e13a2b9193f77153d2785",
        "domain": "numina",
        "question": "1. [20] For how many positive integers $n \\leq 1000$ does the equation in real numbers\n$$\nx^{\\lfloor x\\rfloor}=n\n$$\nhave a positive solution for $x$ ? (For a real number $x,\\lfloor x\\rfloor$ denotes the largest integer that is not greater than $x$.)",
        "answer": "412",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151866,
        "difficulty": null
      },
      {
        "id": "82defdd011afa1932dca6688e8e139a2e608856f89e2239ea7d91ed5e6481b08",
        "domain": "numina",
        "question": "19. On the number line, points $A, B, C, D$ were marked. It is known that the first three points correspond to the numbers 5, 8, and -10. What number corresponds to point $D$, if when the direction of the number line is reversed, the sum of the four numbers corresponding to these points does not change?",
        "answer": "-3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 39596,
        "difficulty": null
      },
      {
        "id": "1aa866307070a88268224c662331f6bc0bc0f4bc7d0a2e80cff936078697126e",
        "domain": "numina",
        "question": "B4 In the following diagram, two lines that meet at a point $A$ are tangent to a circle at points $B$ and $C$. The line parallel to $A C$ passing through $B$ meets the circle again at $D$. Join the segments $C D$ and $A D$. Suppose $A B=49$ and $C D=28$. Determine the length of $A D$.",
        "answer": "63",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 102045,
        "difficulty": null
      },
      {
        "id": "60e1dc751220156558199d5f7e52795cc2b5c7bb4de15aee25edd534c4802682",
        "domain": "numina",
        "question": "11. The Clock Hands Problem. At what time after 12:00 will the minute hand first catch up with the hour hand?",
        "answer": "1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 35606,
        "difficulty": null
      },
      {
        "id": "c7d953b0437f1b5e828eddcfa66bb96376dd21290aad8260e3f286b5a25bb076",
        "domain": "numina",
        "question": "In the game \"What? Where? When?\", 13 envelopes with questions from the audience are played. The envelopes are chosen in a random order using a spinner. If the Experts answer correctly, they earn a point; if incorrectly, one point goes to the audience. The game ends as soon as one of the teams scores 6 points.\n\nAssume that the teams of Experts and Audience are equally matched.\n\na) Find the expected number of points scored by the Experts team in 100 games.\n\nb) Find the probability that in the next game, envelope No. 5 will be played.",
        "answer": "0.715",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 71665,
        "difficulty": null
      },
      {
        "id": "d276ffa140782ade7c862fce87db3cb422ee181bdf780a502e27b17503f7ada4",
        "domain": "numina",
        "question": "Problem 8.2. In the chat of students from one of the schools, a poll was being held: \"On which day to hold the disco: October 22 or October 29?\"\n\nThe graph shows how the votes were distributed an hour after the start of the poll.\n\nThen, 80 more people participated in the poll, voting only for October 22. After that, the poll ended. The final distribution of votes is also shown on the graph.\n\nHow many people participated in the poll?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-24.jpg?height=494&width=747&top_left_y=1295&top_left_x=339)",
        "answer": "260",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 17357,
        "difficulty": null
      },
      {
        "id": "1e763d2dc95e3b0504cf2d850945581d5d3be235cb918fd4aebb1e31375de2fc",
        "domain": "numina",
        "question": "G9.3 If $S=a b+a-b-1$ and $a=101, b=49$, find $S$.",
        "answer": "5000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152556,
        "difficulty": null
      },
      {
        "id": "f6e096c06849931ae935fac2999d985efbb6865250ebd07ec05fb659cff0fa25",
        "domain": "numina",
        "question": "\\section*{Exercise 5 - 031025}\n\nBy which numbers is the product of three arbitrary, but consecutive positive integers divisible, whose sum is odd?",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 184897,
        "difficulty": null
      },
      {
        "id": "62dd27720355d32426e2b25ab67fe20b934721e6c3598bd5dc46ad31e0a67ece",
        "domain": "numina",
        "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $10$, and $\\cos \\alpha = \\frac{4}{5}$?",
        "answer": "16",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 19421,
        "difficulty": null
      },
      {
        "id": "aca6e652bc1db44cd0e8a0345d48aad3e988bf98eb4d4346ecd75c48439d1f90",
        "domain": "numina",
        "question": "[ Ratio of areas of triangles with a common angle ]\n\nOn the sides $A B, B C$ and $A C$ of triangle $A B C$, whose area is 75, points $M, N$ and $K$ are located respectively. It is known that $M$ is the midpoint of $A B$, the area of triangle $B M N$ is 15, and the area of triangle $A M K$ is 25. Find the area of triangle $C N K$.\n\n#",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 59645,
        "difficulty": null
      },
      {
        "id": "41bb4d6d159dbdc265432fde27ecca57b61a3cba430efc130f78456773387367",
        "domain": "numina",
        "question": "6. If the centroid of the inscribed $\\triangle A B C$ of the curve $y^{2}=4 \\sqrt{2} x$ is its focus $F$. Then\n$$\n|F A|^{2}+|F B|^{2}+|F C|^{2}=\n$$\n$\\qquad$",
        "answer": "27",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 137161,
        "difficulty": null
      },
      {
        "id": "c8d997c1a83a6cf4c65123f7813640a14b8c72149c89e53f77f1278064510859",
        "domain": "numina",
        "question": "4. In an acute-angled triangle $A B C$, the altitudes $A A_{1}, B B_{1}$, and $C C_{1}$ are dropped. A point $T$ is chosen on the plane such that the lines $T A$ and $T B$ are tangents to the circumcircle of triangle $A B C$, and point $O$ is the center of this circle. The perpendicular dropped from point $T$ to the line $A_{1} B_{1}$ intersects the line $C C_{1}$ at point $K$, and the line passing through point $C_{1}$ parallel to $O K$ intersects the segment $C O$ at point $L$. Find the angle $\\angle C L A_{1}$.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8303,
        "difficulty": null
      },
      {
        "id": "ed69f29df8ee68cf74089e5f94af4602000f2dff25969e0604709f1df7d7db99",
        "domain": "numina",
        "question": "6. Thirteen identical metal rods are connected as follows (see fig.). It is known that the resistance of one rod \\( R_{0}=10 \\) Ohms. Determine the resistance of the entire structure if it is connected to a current source at points \\( A \\) and \\( B \\). (10 points)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_84fc2054d2075e5d2c9fg-05.jpg?height=537&width=945&top_left_y=154&top_left_x=584)",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4070,
        "difficulty": null
      },
      {
        "id": "99e3dc776cc01138fe95e2329dab7d11f1753f9e5ccc29ec268f7ed9eee0f5b9",
        "domain": "numina",
        "question": "[ Monotonicity, boundedness $\\quad$ ]\n\nFind the maximum value of the expression $a+b+c+d-ab-bc-cd-da$, if each of the numbers $a, b, c$ and $d$ belongs to the interval $[0,1]$.\n\n#",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 67860,
        "difficulty": null
      },
      {
        "id": "51a146e05ea5f3f96611dec29d18779d0c36d16cd4118845b28d6807e07aa6e3",
        "domain": "numina",
        "question": "4. (4 points) Oleg usually arrives on a business trip by the 11 o'clock train. A car arrives at the station for this train. This time, the train arrived an hour earlier, and Oleg started walking towards the car. Meeting the car on the way, he got in, and as a result, arrived 10 minutes earlier than planned. Determine the time Oleg met the car.",
        "answer": "10",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 8618,
        "difficulty": null
      },
      {
        "id": "a6cd0f1cc7aa54c727a5539d74d7d0f580ec276e386b928dbb84d434128203d4",
        "domain": "numina",
        "question": "6. Given that $D$ is the intersection of the tangents to the circumcircle of $\\triangle A B C$ at $A$ and $B$, the circumcircle of $\\triangle A B D$ intersects line $A C$ and segment $B C$ at another point $E$ and $F$ respectively, and $C D$ intersects $B E$ at point $G$. If $\\frac{B C}{B F}=2$, find $\\frac{B G}{G E}$.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127873,
        "difficulty": null
      },
      {
        "id": "eb267218102acdaa8e371919ce7ed91284a4f56cfd6d7d4472ae919af9938d8c",
        "domain": "numina",
        "question": "1. (6 points) Calculate: $2017 \\times \\frac{2015}{2016}+\\frac{1}{2016}=$",
        "answer": "2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 101141,
        "difficulty": null
      },
      {
        "id": "345cac62b5d15aa36cf6c205ae33ae945f013412f881626be0754fb72b73d7d1",
        "domain": "numina",
        "question": "577. a) From a sample of size $n$, extracted from a normal general population with a known standard deviation $\\sigma$, the sample mean $\\bar{x}$ has been found. At a significance level $\\alpha$, it is required to find the power function of the test criterion for testing the null hypothesis $H_{0}: a=a_{0}$ about the equality of the general mean $a$ to the hypothetical value $a_{0}$ under the alternative hypothesis $H_{1}: a=a_{1} \\neq a_{0}$.\n\n**Solution. The alternative hypothesis has the form** $a \\neq a_{0}$, therefore the critical region is two-sided. Using rule 1, we find the critical point $u_{\\text {cr }}$ from the equality $\\Phi\\left(u_{\\text {cr }}\\right)=(1-\\alpha) / 2$. Consequently, the two-sided critical region is defined by the inequality $|U|>u_{\\mathrm{cr}}$, or in more detail\n\n$$\n\\left|\\frac{\\left(\\bar{x}-a_{0}\\right)}{\\sigma / \\sqrt{n}}\\right|>u_{\\mathrm{cr}}\n$$\n\nWe will find the power of the considered criterion, i.e., the probability of the criterion falling into the critical region assuming that the alternative hypothesis $a=a_{1} \\neq a_{0}$ is true:\n\n$$\n1-\\beta=P\\left(\\left|\\frac{x-a_{0}}{\\sigma / \\sqrt{n}}\\right|>u_{\\mathrm{cr}} ; \\quad a=a_{1}\\right)\n$$\n\nTransform the expression under the modulus sign:\n\n$$\n\\frac{\\bar{x}-a_{0}}{\\sigma / \\sqrt{n}}=\\frac{\\bar{x}-a_{1}}{\\sigma / \\sqrt{n}}+\\frac{a_{1}-a_{0}}{\\sigma / \\sqrt{n}}=b+\\lambda\n$$\n\nwhere $b=\\frac{\\bar{x}-a_{1}}{\\sigma / \\sqrt{n}}, \\lambda=\\frac{a_{1}-a_{0}}{\\sigma / \\sqrt{n}}$. Using these relations, we get\n\n$$\n\\begin{gathered}\n1-\\beta=P\\left(|b+\\lambda|>u_{\\mathrm{cr}}\\right)=P\\left(b+\\lambda>u_{\\mathrm{cr}}\\right)+P\\left(b+\\lambda<-u_{\\mathrm{cr}}\\right)= \\\\\n=P\\left(b>u_{\\mathrm{cr}}-\\lambda\\right)+P\\left(b<-u_{\\mathrm{cr}}-\\lambda\\right)= \\\\\n=\\left[1-P\\left(b<u_{\\mathrm{cr}}-\\lambda\\right)\\right]+P\\left(b<-u_{\\mathrm{cr}}-\\lambda\\right)= \\\\\n=\\left[1-\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)\\right]+\\Phi\\left(-u_{\\mathrm{cr}}-\\lambda\\right)=1-\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)-\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right) .\n\\end{gathered}\n$$\n\nThus, the power of the two-sided criterion at $a=a_{1}$ is\n\n$$\n1-\\beta=1-\\left[\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)+\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right)\\right]\n$$\n\nwhere $\\lambda=\\left(a_{1}-a_{0}\\right) \\sqrt{n} / \\sigma$.\n\nEach value of $a_{1}$ corresponds to a certain power value, so the power of the criterion is a function of $a_{1}$; denote it by $\\pi_{2}\\left(a_{1}\\right)$.\n\nThus, the sought power of the two-sided criterion is\n\n$$\n\\pi_{2}\\left(a_{1}\\right)=1-\\left[\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)+\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right)\\right]\n$$\n\nwhere $\\Phi(x)$ is the Laplace function, $\\lambda=\\left(a_{1}-a_{0}\\right) \\sqrt{n} / \\sigma, u_{\\text {cr }}$ is found from the equality $\\Phi\\left(u_{\\text {cr }}\\right)=(1-\\alpha) / 2$.\n\nb) From a sample of size $n=16$, extracted from a normal general population with a known standard deviation $\\sigma=5$, at a significance level of 0.05, the null hypothesis $H_{0}: a=a_{0}=20$ about the equality of the general mean $a$ to the hypothetical value $a_{0}=20$ is tested under the alternative hypothesis $H_{1}: a \\neq 20$. Find the power of the two-sided criterion for testing the considered hypothesis for the hypothetical value of the general mean $a_{1}=24$.",
        "answer": "0.8925",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 32477,
        "difficulty": null
      },
      {
        "id": "c1dd41f7832daa9c79725f184dbd14268f2517dcc06e4d0dcdd6bdcfb03c7645",
        "domain": "numina",
        "question": "30. How many zeros does the product of all integers from 1 to 30 inclusive end with?\n\n## Problems for the sixth grade",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 44612,
        "difficulty": null
      },
      {
        "id": "de68cd18ab7ce821b534b95bd73ada1854db22cdf4e286ffcd6c8e8b5c5c1f66",
        "domain": "numina",
        "question": "1. Foma and Yerema were walking in the same direction along a road with kilometer markers. In one hour, Foma passed by five markers, while Yerema passed by six. Could Foma's speed have been greater than Yerema's?",
        "answer": "5200",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 6849,
        "difficulty": null
      },
      {
        "id": "b213ecb5413ae2f46ef9b4c1ca7fa137af8bacce32526045ebf3b295e32a4b64",
        "domain": "numina",
        "question": "29th IMO 1988 shortlist Problem 20 Find the smallest n such that if {1, 2, ... , n} is divided into two disjoint subsets then we can always find three distinct numbers a, b, c in the same subset with ab = c.",
        "answer": "96",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 148463,
        "difficulty": null
      },
      {
        "id": "3caf463cf87ba565f5c3fad3921b6d41820dc898bc935f9cefc368021f44e4a8",
        "domain": "numina",
        "question": "4. How many elements does the multiplicative group of 2x2 matrices with elements in residue classes modulo 3 have? What about the subgroup of matrices that have a determinant of 1 modulo 3?\n\nSupplement with exercises G.M.1/2014\n\nNOTE: All subjects are mandatory.\n\nEach subject is graded from 0 to 7 points.\n\nWorking time 3 hours.\n\nEach subject should be written on a separate sheet.",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 179097,
        "difficulty": null
      },
      {
        "id": "522ac7ef5d91889c5238f99abe14ae2ddf9d0ab01d37bdf3d5ae789c3cf0b93c",
        "domain": "numina",
        "question": "7. (6 points) The number in the parentheses that satisfies the equation below is $\\qquad$\n$$\n\\frac{\\frac{(}{128}+1 \\frac{2}{7}}{5-4 \\frac{2}{21} \\times 0.75} \\div \\frac{\\frac{1}{3}+\\frac{5}{7} \\times 1.4}{\\left(4-2 \\frac{2}{3}\\right) \\times 3}=4.5\n$$",
        "answer": "\\frac{1440}{7}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 114706,
        "difficulty": null
      },
      {
        "id": "2bbfc2ad41ad2e84e9b8715c698d1d9d159b36c043e408ccf59b1a2a20915d49",
        "domain": "numina",
        "question": "[The product of the lengths of the segments of chords and the lengths of the segments of secants] [Properties and characteristics of an isosceles triangle.] [Thales' theorem and the theorem of proportional segments]\n\nOn the extension of side $A D$ of rhombus $A B C D$ beyond point $D$, point $K$ is taken. Lines $A C$ and $B K$ intersect at point $Q$. It is known that $A K=14$ and that points $A, B$ and $Q$ lie on a circle with radius 6, the center of which lies on segment $A A$. Find $B K$.",
        "answer": "7",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 57796,
        "difficulty": null
      },
      {
        "id": "2c36379eff841e790ab1e989855b08ccc1718f5d4a4477635ee0915b4f1e867a",
        "domain": "numina",
        "question": "Determine the number of convex polygons all of whose sides are the square roots of positive integers which can be inscribed in a unit circle. Polygons that are rotations or reflections of each other are considered the same.",
        "answer": "14",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 90283,
        "difficulty": null
      },
      {
        "id": "1c2dc547d98196f2a1d3da92675f1f15ef3d20c4b3bfca015c3697919d921472",
        "domain": "numina",
        "question": "6. Let the side length of a regular $n$-sided polygon be $a$, and the longest and shortest diagonals be $b$ and $c$ respectively. If $a=b-c$, then $n=$ $\\qquad$",
        "answer": "9",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113967,
        "difficulty": null
      },
      {
        "id": "b919935cc189c75039798e900ce9ac15e56180ffec0a05aab6f5344c31c79509",
        "domain": "numina",
        "question": "## 2. Rectangles\n\nCalculate the sum of all possible different perimeters that rectangles with an area of 144 and whose side lengths are natural numbers can have.\n\nResult: $\\quad 830$",
        "answer": "830",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 89940,
        "difficulty": null
      },
      {
        "id": "7f741253ea077053f24ae498407597b5ca4e64a2ef60dc72a98bb879b6f97779",
        "domain": "numina",
        "question": "1. We will divide the integers from 1 to 9 into three groups of three and then multiply the numbers within each group.\n\na) Determine these three products, given that two of them are equal and smaller than the third product.\n\nb) Assume that one of the three products, denoted as $S$, is smaller than the other two products (which may be the same). Find the largest possible value of $S$.",
        "answer": "70",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 75276,
        "difficulty": null
      },
      {
        "id": "af704e442a4c32fa060cb0ce4c0cc8c17fad0f3de9e15f9638de2e073396d144",
        "domain": "numina",
        "question": "Problem 5.3. Petya and Vasya were playing with toy soldiers. Petya arranged his knights in a \"rectangle\" - some number of columns and some number of rows. When all the knights from the first and second rows went on reconnaissance, 24 knights remained. Then Vasya's archers drove away all the knights who were left in the first and second columns. After this, 18 knights remained. How many knights did Petya have initially?",
        "answer": "40",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13590,
        "difficulty": null
      },
      {
        "id": "4c9f882af3ae971d40c0daa11c5828cc0452f41d496318606aebaab4d614526c",
        "domain": "numina",
        "question": "2. If the three points\n$$\n\\begin{array}{l}\n(1, a, b) \\\\\n(a, 2, b) \\\\\n(a, b, 3)\n\\end{array}\n$$\nare collinear (in 3-space), what is the value of $a+b$ ?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152272,
        "difficulty": null
      },
      {
        "id": "7e36cf475e313fe14e0d1d79d3d3d6fc8b4acd539c66ed64ffec69b861319166",
        "domain": "numina",
        "question": "5. (8 points) Fill in the blanks with $1-6$, so that the numbers in each row and each column are not repeated. The two cells occupied by the same symbol in the figure have the same number combination, but the order is uncertain. Therefore, the five-digit number formed by the first five numbers from left to right in the last row is\n\n保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "46123",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 103320,
        "difficulty": null
      },
      {
        "id": "8366ca6f4ed84fc13a2e86ed8609258b86ece41de07324f091ae488cfca21df6",
        "domain": "numina",
        "question": "# Problem 2.\n\nIn a football tournament, each team is supposed to play one match against each of the others. But during the tournament, half of all the teams were disqualified and withdrew from further participation. As a result, 77 matches were played, and the teams that withdrew had managed to play all their matches among themselves, with each of the withdrawn teams playing the same number of matches. How many teams were there at the beginning of the tournament?",
        "answer": "14",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 19973,
        "difficulty": null
      },
      {
        "id": "68aaa64989cd8c42eced46fd7eb6cc0e57474ee2cdfacb97ad2a7e5297b1f09e",
        "domain": "numina",
        "question": "6. If $f(x)=2 \\sin \\omega x(0<\\omega<1)$ has a maximum value of $\\sqrt{2}$ on the interval $\\left[0, \\frac{\\pi}{3}\\right]$, then $\\omega=$",
        "answer": "\\frac{3}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135589,
        "difficulty": null
      },
      {
        "id": "8fe8a18f004b255020ea82016e0e3936eff5230e632950533e69316fe4ea7f05",
        "domain": "numina",
        "question": "## Task 1 - 250911\n\nFrom the digits 1, 9, 8, 5, all possible four-digit numbers are to be formed, where each of the digits appears exactly once in each of these numbers.\n\nDetermine the number of such numbers that are\n\na) divisible by 2,\n\nb) divisible by 3,\n\nc) divisible by 4,\n\nd) divisible by 5,\n\ne) divisible by 6,\n\nand list these numbers!",
        "answer": "0",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 184675,
        "difficulty": null
      },
      {
        "id": "2e6c8759037c963b595b76733c5a0107b3020bbccf72eaffdb30fbf7a56d6859",
        "domain": "numina",
        "question": "7.5. Egor, Nikita, and Innokentiy took turns playing chess with each other (two play, one watches). After each game, the loser would give up their place at the board to the spectator (there were no draws). In the end, it turned out that Egor participated in 13 games, and Nikita in 27. How many games did Innokentiy play?",
        "answer": "14",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 9833,
        "difficulty": null
      },
      {
        "id": "9c3372532624e436734b0287cb8f3641cfa5c53e11b9f52e2700f946ba4ae8be",
        "domain": "numina",
        "question": "97. For the first half of the journey, the motorcyclist traveled at a speed of 30 km/h, and for the second half - at a speed of 60 km/h. Find the average speed of the motorcyclist.",
        "answer": "40",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 44677,
        "difficulty": null
      },
      {
        "id": "112d28ea382c70cfcc6681a166daf15f782f4a2bfe8ce3b064df750774cde318",
        "domain": "numina",
        "question": "[ [Decimal numeral system ]  [ Problems with constraints ]\n\nFind the number of five-digit numbers in decimal notation that contain at least one digit 8.",
        "answer": "37512",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57330,
        "difficulty": null
      },
      {
        "id": "cf82f4de3dc6a1a130c6629216003584cba13dcb2e8789c50f7292b48fb557e5",
        "domain": "numina",
        "question": "Example 14 (IMO-29 Problem) Let $\\mathbf{N}$ be the set of positive integers, and define the function $f$ on $\\mathbf{N}$ as follows:\n(i) $f(1)=1, f(3)=3$;\n(ii) For $n \\in \\mathbf{N}$, we have $f(2 n)=f(n), f(4 n+1)=2 f(2 n+1)-f(n), f(4 n+3)=$ $3 f(2 n+1)-2 f(n)$.\nFind all $n$ such that $n \\leqslant 1988$ and $f(n)=n$.",
        "answer": "92",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 138847,
        "difficulty": null
      },
      {
        "id": "76dd2bd7893eb6ff0b271c46d224daf5fdc35ce83bc446e1b7dd7841c75de55e",
        "domain": "numina",
        "question": "7. Find the area of triangle $ABC$, the vertices of which have coordinates $A(0,0), B(1424233,2848467), C(1424234,2848469)$. Round your answer to the hundredths.",
        "answer": "0.50",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 8613,
        "difficulty": null
      },
      {
        "id": "58f74f881f6a8dff8fac0b8c424df13f60d0ee3424bd22d6bb3a52cbc8b9d888",
        "domain": "numina",
        "question": "Example 6 For the function $y=(n-1) t^{2}-10 t+10, t \\in(0,4]$, does there exist a positive integer $n$ such that $0<y \\leqslant 30$ always holds?",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 110159,
        "difficulty": null
      },
      {
        "id": "e516c84635d449ae32380ea529208efd6472b00b10299259adc97a7ae13c2972",
        "domain": "numina",
        "question": "9-3-1. The numbers from 1 to 217 are divided into two groups: one group has 10 numbers, and the other has 207. It turns out that the arithmetic means of the numbers in the two groups are equal. Find the sum of the numbers in the group of 10 numbers.",
        "answer": "1090",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12269,
        "difficulty": null
      },
      {
        "id": "47bbe6a6a42e13659843e39ecb1213e36e99dc0b28bc515b09c3e00203141bf6",
        "domain": "numina",
        "question": "11. (10 points) Divide the natural numbers $2, 3, 4, \\cdots, n$ into two groups, satisfying (1) the product of any two numbers in the same group is not in this group; (2) any number and its square are not in the same group. Then the maximum value of $n$ is $\\qquad$",
        "answer": "31",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 126271,
        "difficulty": null
      },
      {
        "id": "88c9154246b6ae827d1938f454607f28c65434de48b4f83028805fda92a0acaf",
        "domain": "numina",
        "question": "Exercise 3. Let $A B C$ be an isosceles triangle such that $\\widehat{A B C}=60^{\\circ}$; it is not known at which vertex $A B C$ is isosceles. Find all possible values of $\\widehat{A C B}$.",
        "answer": "60",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 78030,
        "difficulty": null
      },
      {
        "id": "abad0651e9a4262abb8a908e1435fdeff2afaac00a6bd066f33d02101cfdbe93",
        "domain": "numina",
        "question": "## Task B-4.4.\n\nThe set of all points that satisfy the equation $x^{2}+y^{2}=8|x|+6|y|$ is a curve in the plane. Determine the area of the part of the plane bounded by this curve. How many points $(x, y)$ with integer coordinates lie inside this curve (not on the curve) for which $x y>0$?",
        "answer": "104",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88266,
        "difficulty": null
      },
      {
        "id": "0ed43c330823adbb66fbd5521d64459b8c70571fc79761dac6c3e88a789641dd",
        "domain": "numina",
        "question": "3. (7 points) Rubik made a net of a $3 \\times 3 \\times 3$ cube and marked two points on it - see the image. What will be the distance between these points after Rubik glues the net into a cube?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_06b9f4bfa48aaf5de920g-2.jpg?height=388&width=511&top_left_y=523&top_left_x=1318)",
        "answer": "2\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14935,
        "difficulty": null
      },
      {
        "id": "dab4d25cff0e54b5fdf0e72d5df571f54a02531a68e8838d2baed15971486ea1",
        "domain": "numina",
        "question": "10th Chinese 1995 Problem B1 Four solid balls radii 3, 3, 2, 2 touch each other. What is the radius of the solid ball which touches all four balls?",
        "answer": "\\frac{6}{11}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 148596,
        "difficulty": null
      },
      {
        "id": "40bd15808dc2c3fe49482bea93b9c4bae0ae954c13abcbf4f267c1c5b22c3218",
        "domain": "numina",
        "question": "Example 9 Find the smallest positive integer $n(n>1)$, such that $\\sqrt{\\frac{1}{n} \\sum_{i=1}^{n} i^{2}}$ is an integer.\n(2017, Singapore Mathematical Olympiad)",
        "answer": "337",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 124101,
        "difficulty": null
      },
      {
        "id": "6786c2d254a2bb7ea919d7c5910fdfea3c23ce26f81fb36108fbb7ba49c6b076",
        "domain": "numina",
        "question": "10. Find the sum of the roots of the equation param1 that lie in the interval param2. Write the answer in degrees.\n\n| param 1 | param2 |  |\n| :---: | :---: | :---: |\n| $\\sin x+\\sin ^{2} x+\\cos ^{3} x=0$ | $\\left[360^{\\circ} ; 720^{\\circ}\\right]$ |  |\n| $\\cos x-\\cos ^{2} x-\\sin ^{3} x=0$ | $\\left[180^{\\circ} ; 540^{\\circ}\\right]$ |  |\n| $\\sin x+\\sin ^{2} x+\\cos ^{3} x=0$ | $\\left[-360^{\\circ} ; 0^{\\circ}\\right]$ |  |\n| $\\cos x-\\cos ^{2} x-\\sin ^{3} x=0$ | $\\left[-540^{\\circ} ;-180^{\\circ}\\right]$ |  |",
        "answer": "-360",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3347,
        "difficulty": null
      },
      {
        "id": "4af879ed1e8c1be335883a35f17f521a01d35306cb24990bb8610da7cbdd9cba",
        "domain": "numina",
        "question": "1. Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy:\n$$\na_{1}=\\frac{1}{2}, a_{n+1}=\\frac{2 a_{n}}{3 a_{n}+2}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen the value of $a_{2017}$ is $\\qquad$ .",
        "answer": "\\frac{1}{3026}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 102315,
        "difficulty": null
      },
      {
        "id": "76a24b7d0924b1c08c610158e9810def4e876f1ba33836819672bcef4e7fb33f",
        "domain": "numina",
        "question": "$7.43 \\quad 5^{x} \\sqrt[x]{8^{x-1}}=500$.\n\n$7.44 \\quad 5^{\\frac{x}{\\sqrt{x}+2}} \\cdot 0.2^{\\frac{4}{\\sqrt{x}+2}}=125^{x-4} \\cdot 0.04^{x-2}$.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49781,
        "difficulty": null
      },
      {
        "id": "7bcfdf65895904ef613ad5eb75852c98a1b3397d1d993f8260f040d9e04f4b3e",
        "domain": "numina",
        "question": "Find the inverse of 42 modulo 43, then modulo 59\n\nTo verify if needed:",
        "answer": "42",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 84603,
        "difficulty": null
      },
      {
        "id": "1124c11c9c364d8198ce14aacb13332716fe1c7c2e9fc6cc9fff051b7e8d56be",
        "domain": "numina",
        "question": "Let $A B C D$ be a quadrilateral with $\\angle B A D=\\angle A B C=90^{\\circ}$, and suppose $A B=B C=1, A D=2$. The circumcircle of $A B C$ meets $\\overline{A D}$ and $\\overline{B D}$ at points $E$ and $F$, respectively. If lines $A F$ and $C D$ meet at $K$, compute $E K$.",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151556,
        "difficulty": null
      },
      {
        "id": "e85671ee38b5091a7aae1748a59edb3099c6fad0339e70da087f54fec4d7105f",
        "domain": "numina",
        "question": "## Task A-1.5.\n\nKarlo and Lovro are playing the following game. Karlo will cut a paper of size $9 \\times 9$ into rectangles of integer dimensions, where at least one dimension is 1. After that, Lovro will choose a natural number $k \\in \\{1, \\ldots, 9\\}$, and Karlo will give him as many coins as the total area of all rectangles of dimensions $1 \\times k$ and $k \\times 1$. Lovro will choose $k$ to maximize the number of coins he receives, while Karlo wants to save and give Lovro as few coins as possible.\n\nDetermine the smallest possible number of coins that Karlo will give to Lovro.",
        "answer": "12",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 87240,
        "difficulty": null
      },
      {
        "id": "ed7331bb3fac45a10c08614e8855200dbc059ef2056f00b8114a1c03cae3ff6f",
        "domain": "numina",
        "question": "In triangle $ABC$, the coordinates of two vertices are $A(-1 ; 1), \\quad B(1 ; 2)$. The third vertex $C$ is on the $x$-axis. The centroid of the triangle is equidistant from the two coordinate axes.\n\nWhat is the area of the triangle?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 165593,
        "difficulty": null
      },
      {
        "id": "bdaca1b3ce59e7e0c13f970101558c055e825f3cecb49faeef27d8b284f58fa6",
        "domain": "numina",
        "question": "$\\underline{\\text { by Banov I.S }}$\n\nThe height and the radius of the base of a cylinder are equal to 1. What is the smallest number of spheres with a radius of 1 that can completely cover this cylinder?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66874,
        "difficulty": null
      },
      {
        "id": "795a370f193d46247225b05929521a1c450b33889d8a7283b98f73cd0a34abc0",
        "domain": "numina",
        "question": "1. 25 Find all natural numbers $x$ that satisfy the following conditions: the product of the digits of $x$ equals $44x - 86868$, and the sum of the digits is a perfect cube.",
        "answer": "1989",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 125938,
        "difficulty": null
      },
      {
        "id": "9c84ecb3cff53df0623cab0397ed5a1fc1ba72378d140801e159a07e1c6edcfa",
        "domain": "numina",
        "question": "6. At the entrance of Sheep Village, there are three lights - yellow, red, and blue (the positions of the lights cannot be changed), used to indicate signals. For example, \"yellow light on, red light off, blue light on\" is one signal, and \"yellow light off, red light off, blue light off\" is another signal. There are $\\qquad$ such signals.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 102183,
        "difficulty": null
      },
      {
        "id": "46cda35f072cdd963b6999f7f5d9a63af0cba2e928ba2a6884523cb4e93fc2f0",
        "domain": "numina",
        "question": "3. The letters of the word MATHEMATICS are rearranged to form distinct strings of the same 11 letters. What proportion of these strings do not contain the string MATH?",
        "answer": "\\frac{491}{495}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154619,
        "difficulty": null
      },
      {
        "id": "463477bd61ea2d04ece46bbd522115236b9ecd6065e0c696a03995b78260eec1",
        "domain": "numina",
        "question": "8. Given $\\alpha, \\beta, \\gamma \\in(0, \\pi)$, and $\\alpha+\\beta+2 \\gamma=\\pi$, then the maximum value of $\\cos \\alpha+\\cos \\beta+\\sin 2 \\gamma$ is $\\qquad$",
        "answer": "\\frac{3\\sqrt{3}}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 133405,
        "difficulty": null
      },
      {
        "id": "0d86dfe0b802ce9074d4acebc4eabc3a121fc402aa7861cc8471aa7eeb8e4e11",
        "domain": "numina",
        "question": "7.187. $x^{2} \\cdot \\log _{x} 27 \\cdot \\log _{9} x=x+4$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 48362,
        "difficulty": null
      },
      {
        "id": "ce35a5e308f922b7c39e115df37bae9dbaf592ec49450772470d806baa6c0703",
        "domain": "numina",
        "question": "6. (15 points) For a natural number $N$, if at least six numbers in the nine natural numbers from $1 \\sim 9$ can be factors of $N$, then $N$ is called a “Six-Union Number”. Among the natural numbers greater than 2000, the smallest “Six-Union Number” is $\\qquad$",
        "answer": "2016",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 146278,
        "difficulty": null
      },
      {
        "id": "9a258c77d0924b205e4d1a46a470511385a50dbf7da331e8cba75080f2168ea3",
        "domain": "numina",
        "question": "Shapovalov A.V.\n\nThree square paths with a common center are spaced 1 m apart from each other (see figure). Three ants start simultaneously from the lower left corners of the paths and run at the same speed: Mu and Ra counterclockwise, and Wei clockwise. When Mu reaches the lower right corner of the largest path, the other two, not yet having completed a full circle, are on the right sides of their paths, and all three are on the same straight line. Find the sides of the squares.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1aa9346639cd54e59f2ag-06.jpg?height=683&width=640&top_left_y=1202&top_left_x=709)",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 68128,
        "difficulty": null
      },
      {
        "id": "1262fd1255329e01e463e25a6087e72dcd13ae6cbfdbc940e310f1478f5c2570",
        "domain": "numina",
        "question": "Example 7. Find $\\lim _{x \\rightarrow 0} \\frac{\\sin ^{2} \\frac{x}{3}}{x^{2}}$.",
        "answer": "\\frac{1}{9}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32598,
        "difficulty": null
      },
      {
        "id": "1b0aea0040a15284e8cf4ae97b06f6244eefcb76ab8689a89e58bd59e8422886",
        "domain": "numina",
        "question": "1. It is known that $\\sin x=\\frac{3}{2} \\sin y-\\frac{2}{3} \\cos y, \\cos x=\\frac{3}{2} \\cos y-\\frac{2}{3} \\sin y$. Find $\\sin 2 y$.",
        "answer": "\\frac{61}{72}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3032,
        "difficulty": null
      },
      {
        "id": "9c4369818ba9dc50ce6b10996e0177c1cc8ab3337c4672ebc9a740367b9a24a9",
        "domain": "numina",
        "question": "10. Among $1,2,3,4, \\cdots, 1000$, the numbers that can be written in the form $a^{2}-b^{2}+1(a, b \\in \\mathbf{N})$ and are not divisible by 3 are $\\qquad$ in number.",
        "answer": "501",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 112396,
        "difficulty": null
      },
      {
        "id": "9ee2ad68843dcbb98a4ec6565935329da548ffb53c73da69f0a5990511e7a687",
        "domain": "numina",
        "question": "3. We will call a number \"remarkable\" if it has exactly 4 distinct natural divisors, and among them, there are two such that neither is a multiple of the other. How many \"remarkable\" two-digit numbers exist?",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 5554,
        "difficulty": null
      },
      {
        "id": "2e8b2aba2f21825dbaecd8f4091e555fadb32f692b316f9cbce4701667349b70",
        "domain": "numina",
        "question": "8. It is known that two engineering teams, Team A and Team B, have several people each. If 90 people are transferred from Team A to Team B, then the total number of people in Team B will be twice that of Team A; if some people are transferred from Team B to Team A, then the total number of people in Team A will be 6 times that of Team B. What is the minimum number of people originally in Team A?",
        "answer": "153",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145132,
        "difficulty": null
      },
      {
        "id": "8a8c6be4d265f66f1b569b716a6493b9ec3a94fa3f3f47318c1e4103617886f2",
        "domain": "numina",
        "question": "1. If $a b c d=1$, then\n$$\n\\frac{1}{1+a+a b+a b c}+\\frac{1}{1+b+b c+b c d}+\\frac{1}{1+c+c d+c d a}+\\frac{1}{1+d+d a+d a b}=\n$$\n$\\qquad$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 96875,
        "difficulty": null
      },
      {
        "id": "fc9b39563597315600da97517c5a5cf6dd8c96b7bc40780619ac351a0a459a35",
        "domain": "numina",
        "question": "We have selected the vertices, the centroid, and the points that trisect the sides of an equilateral triangle. How many points can we retain from among them so that no three form an equilateral triangle?",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 170575,
        "difficulty": null
      },
      {
        "id": "c36738f626498d899fd3921ee5464e2fb92e9d54dfe35f7f00678c02d8895583",
        "domain": "numina",
        "question": "4. Find the sum\n\n$$\n\\log \\operatorname{tg} 1^{\\circ}+\\log \\operatorname{tg} 2^{\\circ}+\\ldots+\\log \\operatorname{tg} 89^{\\circ}\n$$",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 190739,
        "difficulty": null
      },
      {
        "id": "a02a346edf1a8e82fc91e6bdeb1b2ee2bfc6f3458dbcf90b5817c0447c0c8b11",
        "domain": "numina",
        "question": "VI OM - II - Task 3\n\nWhat should be the angle at the vertex of an isosceles triangle so that a triangle can be constructed with sides equal to the height, base, and one of the remaining sides of this isosceles triangle?",
        "answer": "106",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 690,
        "difficulty": null
      },
      {
        "id": "d0acde24922d92b2d44acc88749187edc88014019202b94c53efea1a3883512d",
        "domain": "numina",
        "question": "11. Let $P$ be any point on the ellipse $\\frac{x^{2}}{16}+\\frac{y^{2}}{9}=1$ other than the endpoints of the major axis, $F_{1}$ and $F_{2}$ be its left and right foci respectively, and $O$ be the center. Then $\\left|P F_{1}\\right| \\cdot\\left|P F_{2}\\right|+|O P|^{2}=$ $\\qquad$ .",
        "answer": "25",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 104116,
        "difficulty": null
      },
      {
        "id": "fc9f60eb67c18a82c1fb775fad4821179e5d071cc0c1f1d515294f38463a8f4a",
        "domain": "numina",
        "question": "9.5 The numbers $x$, $y$, and $z$ satisfy the equations\n\n$$\nx y + y z + z x = x y z, \\quad x + y + z = 1\n$$\n\nWhat values can the sum $x^{3} + y^{3} + z^{3}$ take?",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17520,
        "difficulty": null
      },
      {
        "id": "383bdba4fb188f6cd8bae1b0541860b2306b3671be893702f57b7dff895e6de2",
        "domain": "numina",
        "question": "7.054. $\\log _{2} x+\\log _{4} x+\\log _{8} x=11$.",
        "answer": "64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50826,
        "difficulty": null
      },
      {
        "id": "a680fe8dd3af7b1dcabea3c632a9cadc6409158eee61befed6334ce39b19a744",
        "domain": "numina",
        "question": "Example 11 (25th IMO Shortlist) Find the last two digits of $\\left[(\\sqrt{29}+\\sqrt{21})^{1984}\\right]$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "71",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 142299,
        "difficulty": null
      },
      {
        "id": "ae3c9478c2c6640a74706f16f60a665d7c61fd795482650750220dfddd698412",
        "domain": "numina",
        "question": "The base of an isosceles triangle is $4 \\sqrt{2}$, and the median drawn to the lateral side is 5.\n\nFind the lateral sides.",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 64460,
        "difficulty": null
      },
      {
        "id": "6ca590dab7a40593131fbd516a853d908fc0e3e4f178c706fab1c7a7d08921b1",
        "domain": "numina",
        "question": "6. Given that the base of the tetrahedron $S-ABC$ is an equilateral triangle, the projection of point $A$ on the side face $SBC$ is $H$, which is the orthocenter of $\\triangle SBC$, the dihedral angle $H-AB-C$ is $30^{\\circ}$, and $SA=2\\sqrt{3}$. Then the volume of the tetrahedron $S-ABC$ is",
        "answer": "\\frac{9}{4}\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133065,
        "difficulty": null
      },
      {
        "id": "9654957a3ec76d23b07134db31d5197985eb58b2d50e77733cd39e12f906706f",
        "domain": "numina",
        "question": "13. Ring Line (from 8th grade. 3 points). On weekdays, the Absent-Minded Scientist travels to work on the ring line of the Moscow metro from the station \"Taganskaya\" to the station \"Kievskaya\", and back in the evening (see the diagram).\n\nUpon entering the station, the Scientist boards the first train that arrives. It is known that trains run at approximately equal intervals in both directions, and that the train traveling on the northern route (through \"Belorusskaya\") takes 17 minutes to travel from \"Kievskaya\" to \"Taganskaya\" or vice versa, while the train on the southern route (through \"Paveletskaya\") takes 11 minutes.\n\nOut of habit, the Scientist always calculates everything. Once, he calculated that over many years of observation:\n\n- the train traveling counterclockwise arrives at \"Kievskaya\" on average 1 minute and 15 seconds after the train traveling clockwise arrives at the same station. The same is true for \"Taganskaya\";\n\n- the average travel time from home to work is 1 minute less than the travel time from work to home.\n\nFind the expected interval between trains traveling in the same direction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b695f5b0d6056980ce62g-08.jpg?height=785&width=785&top_left_y=1606&top_left_x=1138)",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 10896,
        "difficulty": null
      },
      {
        "id": "99ef9dbcbaed56f2d700778dc72cc2c8dcd1a2e108ac9035f9b2b2ad20e9983c",
        "domain": "numina",
        "question": "1703. Determine whether Chebyshev's theorem is applicable to the sequence of independent random variables $X_{1}, X_{2}, \\ldots$, $X_{n}, \\ldots$, if $X_{n}$ has the following distribution:\n\n| $x$ | $-5 n$ | 0 | $5 n$ |\n| :---: | :---: | :---: | :---: |\n| $p(x)$ | $\\frac{1}{3 n^{2}}$ | $1-\\frac{2}{3 n^{2}}$ | $\\frac{1}{3 n^{2}}$ |",
        "answer": "\\frac{50}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 30673,
        "difficulty": null
      },
      {
        "id": "7c2c4e4a3bd0b2fe78c3b59bc855249b119bcddc4f6fd556d86e4ddb1f29acb8",
        "domain": "numina",
        "question": "One interior angle in a triangle measures $50^{\\circ}$. What is the angle between the bisectors of the remaining two interior angles?\n\n(L. Hozová)\n\nHint. You don't need to know the sizes of the remaining interior angles to solve the problem.",
        "answer": "65",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 76900,
        "difficulty": null
      },
      {
        "id": "46d98eb4437004c5ebd86ebaa8fac11d75550a7425cb27230ac191d5495ce556",
        "domain": "numina",
        "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15957,
        "difficulty": null
      },
      {
        "id": "95f416ba88745ecc4b23a2cb103a949bf68a56927562f898f1e9c072d76d9d10",
        "domain": "numina",
        "question": "4. In the maze diagram in Fig. 1, each segment (link) is a corridor, and each circle is a small room. In some rooms, there are beacons that hum - each with its own voice. When in any room, the robot hears the signal from each beacon and determines the distance to it by the attenuation, i.e., the number of links on the shortest path to the beacon. For example, if a beacon is located in room $A$ (in chess notation, this is room a4), and the robot is in room $B$ (this is room g4), the distance between them is 7 links. The robot has the maze diagram and the beacons are marked on it. For what minimum number of beacons can the robot, when in any room, uniquely determine its location by sound?\n\na) Indicate how such a number of beacons can be placed. Why will the robot be able to determine its room by them?\n\nb) Prove that it is impossible to manage with a smaller number of beacons.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_57b8ad17478f0226d916g-02.jpg?height=505&width=529&top_left_y=513&top_left_x=775)\n\nFig. 1: Maze",
        "answer": "3",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 8283,
        "difficulty": null
      },
      {
        "id": "729a7e66f5e458a24376aa158b656424d795d5cbcf4bd761b29727461137a3ca",
        "domain": "numina",
        "question": "Shveiov D.V.\n\nThe circle inscribed in the right triangle $ABC\\left(\\angle ABC=90^{\\circ}\\right)$ touches the sides $AB, BC, AC$ at points $C_{1}, A_{1}, B_{1}$ respectively. The excircle opposite to $A$ touches the side $BC$ at point $A_{2}$. $A_{0}$ is the center of the circumcircle of triangle $A_{1} A_{2} B_{1}$; the point $C_{0}$ is defined similarly. Find the angle $A_{0} B C_{0}$.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61037,
        "difficulty": null
      },
      {
        "id": "c96c25f127c751b2bc08dd408e2b22b67e5b7cdd39c51d78540d00b7c9889a37",
        "domain": "numina",
        "question": "5. Given that $f(x)$ and $g(x)$ are two quadratic functions with the leading coefficient of 1, if $g(6)=35, \\frac{f(-1)}{g(-1)}=$ $\\frac{f(1)}{g(1)}=\\frac{21}{20}$, then $f(6)=$ $\\qquad$ .",
        "answer": "35",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 139825,
        "difficulty": null
      },
      {
        "id": "b7242fd59b5d39cd37f24cd16aca413c174996fce815816ee34dbce972b226c5",
        "domain": "numina",
        "question": "8. There are some chess pieces in a box, with the quantity being less than 50. Xiao Ming and Xiao Liang take turns to take chess pieces from the box. If they take the pieces in the pattern of Xiao Ming taking 2, Xiao Liang taking 2, Xiao Ming taking 2, and Xiao Liang taking 2, in the end, Xiao Ming takes 2 more pieces than Xiao Liang; If they take the pieces in the pattern of Xiao Ming taking 3, Xiao Liang taking 3, Xiao Ming taking 3, and Xiao Liang taking 3, in the end, both of them take the same number of pieces. Therefore, the box contains at most $\\qquad$ pieces.",
        "answer": "42",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 113130,
        "difficulty": null
      },
      {
        "id": "a7eda36b2c7b46d0c3c0cf5446e5c06565bebbd6c8f61b2ccc2e02124d6269df",
        "domain": "numina",
        "question": "Zeljko travelled at $30 \\mathrm{~km} / \\mathrm{h}$ for 20 minutes and then travelled at $20 \\mathrm{~km} / \\mathrm{h}$ for 30 minutes. How far did he travel, in kilometres?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194397,
        "difficulty": null
      },
      {
        "id": "3f7b9de0276829381e2b6b2206d6fdbdeb2b802915ab267f09b41f193cd58a60",
        "domain": "numina",
        "question": "2. (5 points) The same remote control plane is sold online for 300 yuan, which is $20 \\%$ lower than the price at Star Toy Store. Therefore, the price of this remote control plane at Star Toy Store is $\\qquad$ yuan.",
        "answer": "375",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103243,
        "difficulty": null
      },
      {
        "id": "c09969c8cd9d1cdb4592042fa2c9187c01cb90ca26e1bfc40f7c69fb623b5283",
        "domain": "numina",
        "question": "6. As shown in Figure 1, in the tetrahedron\n$$\n\\begin{array}{l}\nA B C D \\text {, } D A=D B=D C \\\\\n=2, D A \\perp D B, D A \\perp\n\\end{array}\n$$\n$D C$, and the cosine of the angle between $D A$ and the plane $A B C$ is $\\frac{\\sqrt{6}}{3}$. Then the radius $R$ of the circumscribed sphere of the tetrahedron is $\\qquad$",
        "answer": "\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 124549,
        "difficulty": null
      },
      {
        "id": "d438685e0d637f228d45a0419d82af1c3deffe986805700e8d296222be1692ad",
        "domain": "numina",
        "question": "# 1. Clone 1\n\nThe figure is divided into 7 equal squares and several rectangles. The perimeter of rectangle A is 112 cm. What is the perimeter of rectangle B? Express your answer in centimeters.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f50971f08007f421b41g-01.jpg?height=485&width=483&top_left_y=1008&top_left_x=838)\n\n#",
        "answer": "168",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 14714,
        "difficulty": null
      },
      {
        "id": "96e0f22a715c48822fa5498cbb0baf0fddfd78ea91aa7beb09f81a4c8cbdf60e",
        "domain": "numina",
        "question": "SI. 4 In the coordinate plane, the distance from the point $(-C, 0)$ to the straight line $y=x$ is $\\sqrt{D}$, find the value of $D$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153172,
        "difficulty": null
      },
      {
        "id": "0894e86bcc069a421b419d4fe6c65dbffa19f96613bd49132ca7130a6aa359a3",
        "domain": "numina",
        "question": "Problem 10.7. The graph of the function $f(x)=\\frac{1}{12} x^{2}+a x+b$ intersects the $O x$ axis at points $A$ and $C$, and the $O y$ axis at point $B$, as shown in the figure. It turned out that for the point $T$ with coordinates $(3 ; 3)$, the condition $T A=T B=T C$ is satisfied. Find $b$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_df4c0436ab68970fab15g-39.jpg?height=321&width=517&top_left_y=195&top_left_x=468)",
        "answer": "-6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 16321,
        "difficulty": null
      },
      {
        "id": "3ce882cc04e13c97f6bfb91ab7794b60b9ca76012cb7ce48a648e57cd9e9160f",
        "domain": "numina",
        "question": "5. Given that $a$, $b$, and $c$ are all positive numbers. Then\n$$\nM=\\min \\left\\{\\frac{1}{a}, \\frac{2}{b}, \\frac{4}{c}, \\sqrt[3]{a b c}\\right\\}\n$$\n\nthe maximum value of $M$ is . $\\qquad$",
        "answer": "\\sqrt{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 136367,
        "difficulty": null
      },
      {
        "id": "aeadfa0898ad10538b5bf8ed7ddf1139e118bf3158ed87c1a3a2a078526cc2e2",
        "domain": "numina",
        "question": "7. The diagonals of a trapezoid are perpendicular to each other, and one of them is 17. Find the area of the trapezoid if its height is 15.\n\nAnswer. $\\frac{4335}{16}$.",
        "answer": "\\frac{4335}{16}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22991,
        "difficulty": null
      },
      {
        "id": "6695ac66cc342142658826902639d57e35816598a3731973c894fa28cf97d439",
        "domain": "numina",
        "question": "How many positive integers of at most $n$ digits are there whose square ends in the same $n$ digits (possibly with some leading zeros)?",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171007,
        "difficulty": null
      },
      {
        "id": "fa1d7a4c9f1bfc0ea2559b0de33486dbf5e9b0a93f2a213f7fbddcdedc2c40d4",
        "domain": "numina",
        "question": "3. Given that the length of the major axis of an ellipse is 4, the focal distance is 2, and the sum of the lengths of two perpendicular chords passing through the left focus is $\\frac{48}{7}$. Try to find the product of the lengths of these two chords.\n\n untranslated text remains unchanged.",
        "answer": "\\frac{576}{49}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 108539,
        "difficulty": null
      },
      {
        "id": "e32dd8fd9fdd40c3e4d80f5dfc51993107d4d2bb316a2950b545c8b8e11768e8",
        "domain": "numina",
        "question": "Variant 10.4.1. Given a parallelogram $A B C D$, point $M$ is the midpoint of side $B C$. On side $A D$, there is a point $K$ such that $B K=B M$ and quadrilateral $K B M D$ is cyclic.\n\n(a) (2 points) What is the length of segment $M D$, if $A D=17$?\n\n(b) (2 points) How many degrees does angle $K M D$ measure, if $\\angle B A D=46^{\\circ}$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_baaf26a33e2d9a36d9b4g-10.jpg?height=274&width=517&top_left_y=824&top_left_x=468)",
        "answer": "8.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13119,
        "difficulty": null
      },
      {
        "id": "081645e3b79edd1244f62819e44c025bbe254f07d81324a14f990dedc2242625",
        "domain": "numina",
        "question": "28. [14] Compute the nearest integer to\n$$\n100 \\sum_{n=1}^{\\infty} 3^{n} \\sin ^{3}\\left(\\frac{\\pi}{3^{n}}\\right)\n$$",
        "answer": "236",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 151939,
        "difficulty": null
      },
      {
        "id": "5dfef286c4e7c118008aee8da0af3588ff4ca0c1d43129da125ca5c841bd06ea",
        "domain": "numina",
        "question": "14. (3 points) The unit digit of $1^{8}+2^{8}+3^{8}+4^{8}+5^{8}+6^{8}+7^{8}+8^{8}+9^{8}+10^{8}+11^{8}$ is",
        "answer": "4",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 118776,
        "difficulty": null
      },
      {
        "id": "b487c323bb618892beeb8bf8382160c62986fb93138a1017f7a6e7c37fd6fd30",
        "domain": "numina",
        "question": "## 298. Math Puzzle $3 / 90$\n\nSebastian visits his grandmother every week, who lives in the neighboring town 10 km away. Most of the time, he rides his bicycle. On a nice day, it takes the grandmother's favorite grandson 1 hour and 20 minutes to get to the destination and back.\n\nOne day, however, a strong wind is blowing. The boy still sets off on his bike.\n\nThe speed increases by $4 \\mathrm{~km} / \\mathrm{h}$ on the way there, and decreases by the same amount on the way back. Sebastian thinks that this does not change his travel time? Is he right?",
        "answer": "86.1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 181964,
        "difficulty": null
      },
      {
        "id": "1a783e6813938f0ec0be4690d76bfd4216aad7a01ea67b66e87ee86bed2b3552",
        "domain": "numina",
        "question": "8.5. Given a convex quadrilateral $A B C D$, where $A B=A D=1, \\angle A=80^{\\circ}$, $\\angle C=140^{\\circ}$. Find the length of the diagonal $A C$.",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 22523,
        "difficulty": null
      },
      {
        "id": "ab7228db0dac0a4aaca9fbb79f9ed6e8a2c7fd2d4e9521cdd090eee854d02064",
        "domain": "numina",
        "question": "The Law of Cosines\n\nThe bases of a trapezoid are 3 cm and 5 cm. One of the diagonals of the trapezoid is 8 cm, and the angle between the diagonals is $60^{\\circ}$. Find the perimeter of the trapezoid.",
        "answer": "22",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63499,
        "difficulty": null
      },
      {
        "id": "d1f39dfb96eae441ba5664f236ca84fca233a4dbf4d690f77d32d28825238d64",
        "domain": "numina",
        "question": "Find all integers $n$ such that $n^{3}-3 n^{2}+n+2$ is a power of 5.",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 84632,
        "difficulty": null
      },
      {
        "id": "280842c779cb4a8cb48187cd20ae2780c05271ad2095bb9d3ae15f464738ed84",
        "domain": "numina",
        "question": "The table below shows some of the results from the last Pirajuba Fishing Festival, displaying how many competitors $q$ caught $n$ fish for some values of $n$.\n\n| $n$ | 0 | 1 | 2 | 3 | $\\ldots$ | 13 | 14 | 15 |\n| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |\n| $q$ | 9 | 5 | 7 | 23 | $\\ldots$ | 5 | 2 | 1 |.\n\nThe news published in the city's newspaper reported that:\n\ni) the winner caught 15 fish;\n\nii) among those who caught 3 or more fish, the average was 6 fish caught; and\n\niii) among those who caught 12 or fewer fish, the average number of fish caught was 5.\n\na) What was the total number of fish caught during the festival?\n\nb) How many competitors caught between 4 and 12 fish?",
        "answer": "883",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73363,
        "difficulty": null
      },
      {
        "id": "7e51e5573e2635c7eabf0b2583352c0217c9391ecf2a35e44d900ad5a53c1654",
        "domain": "numina",
        "question": "Task B-4.1. Determine the natural number $n \\geq 2$ such that the equality\n\n$$\n\\frac{(n-1)^{2} n(n+1)!}{(n+2)!}=\\binom{n}{2}\n$$\n\nholds.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 88963,
        "difficulty": null
      },
      {
        "id": "7c2789ada087cd57aeeb0101236bf33eb6b19e8cfb945ccfadf58ef1b8396429",
        "domain": "numina",
        "question": "Let $t$ be TNYWR.\n\nWhat is the area of the triangle enclosed by the $x$-axis, the $y$-axis, and the line with equation $y=t x+6$ ?\n\n## The CENTRE for EDUCATION in MATHEMATICS and COMPUTING cemc.uwaterloo.ca 2022 Canadian Team Mathematics Contest",
        "answer": "\\frac{9}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194389,
        "difficulty": null
      },
      {
        "id": "cf50f0f48ddaea1ce5aa9db0cc1594bb3f2b9dfa4a144efd9012d53ecbae2f52",
        "domain": "numina",
        "question": "10. From the numbers $1,2,3, \\cdots, 14$, select $a_{1}, a_{2}$, $a_{3}$ in ascending order, and $a_{2}-a_{1} \\geqslant 3, a_{3}-a_{2} \\geqslant 3$, then the number of different ways to select that meet the conditions is $\\qquad$ kinds.",
        "answer": "120",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 91879,
        "difficulty": null
      },
      {
        "id": "62da303497ee74af51b8adaeb5bb07a78fe92566e0545c42f828c7c9eb5efeaf",
        "domain": "numina",
        "question": "## Task B-3.1.\n\nDetermine the maximum value of the function $f(x)=\\log _{2}^{4} x+12 \\log _{2}^{2} x \\cdot \\log _{2} \\frac{8}{x}$ on the interval $[1,64]$.",
        "answer": "81",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89523,
        "difficulty": null
      },
      {
        "id": "52bf7481a5f2956243639749927cb3f8c7a778289e6942477ea4d134ad13f609",
        "domain": "numina",
        "question": "$9 \\cdot 8$ For all positive integers from 1 to 1000000000, find the sum of the digits of each number, then for the resulting 1 billion numbers, find the sum of the digits of each number, $\\cdots$, until obtaining 1 billion single-digit numbers. Ask whether there are more 1s or 2s in the resulting numbers?\n(4th All-Russian Mathematical Olympiad, 1964)",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 123138,
        "difficulty": null
      },
      {
        "id": "a7e0f7368cfbf82cd0713257fdc3e5db540dd6c431261a0466255a70ab63c814",
        "domain": "numina",
        "question": "4. The sequence $\\left(x_{n}\\right)_{n=1}^{\\infty} \\mathrm{s}$ has the first term $x_{1}=1$ and for every $n>1$ satisfies the condition\n\n$$\nx_{n}=x_{n-1}^{ \\pm 1}+x_{n-2}^{ \\pm 1}+\\ldots+x_{1}^{ \\pm 1}\n$$\n\nwith an appropriate choice of the signs \"±\" in the exponents of the powers.\n\na) Determine whether any term of such a sequence must be greater than 1000.\n\nb) Find the smallest possible value of the term $x_{1000000}$.\n\nc) Prove that the inequality $x_{n}<4$ cannot hold for nine terms $x_{n}$ of such a sequence.\n\nII. round of category A takes place\n\n## on Tuesday, January 14, 2003\n\nso that it starts in the morning and the contestants have 4 hours of pure time to solve the problems. Each problem can earn the contestant 6 points, and a successful solver is a student who scores 10 points or more. These details will be communicated to the students before the start of the competition.",
        "answer": "3-2^{3-10^{6}}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 75456,
        "difficulty": null
      },
      {
        "id": "ffdb949951bef32c3b09eb8318c17505d610b0a0cc3db843df38fd489933bba8",
        "domain": "numina",
        "question": "[ Right triangle with an angle in ]\n\nThe hypotenuse $AB$ of the right triangle $ABC$ is 2 and is a chord of a certain circle. The leg $AC$ is 1 and lies inside the circle, and its extension intersects the circle at point $D$, with $CD=3$. Find the radius of the circle.",
        "answer": "2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62298,
        "difficulty": null
      },
      {
        "id": "1d2ec89fbc80726450b5a53bae80d7f1c73099d6bab2b7f2a62d803ca873df81",
        "domain": "numina",
        "question": "2. On the first day, Masha collected $25 \\%$ fewer berries than Vanya, and on the second day, $20 \\%$ more than Vanya. Over the two days, Masha collected $10 \\%$ more berries than Vanya. What is the smallest number of berries they could have collected together?",
        "answer": "189",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 13091,
        "difficulty": null
      },
      {
        "id": "1487424e29903d4e481fa14185eae9e69115bfb185670794fe60c3fe70c814d0",
        "domain": "numina",
        "question": "1.005. $\\frac{2 \\frac{3}{4}: 1.1+3 \\frac{1}{3}}{2.5-0.4 \\cdot 3 \\frac{1}{3}}: \\frac{5}{7}-\\frac{\\left(2 \\frac{1}{6}+4.5\\right) \\cdot 0.375}{2.75-1 \\frac{1}{2}}$.",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50335,
        "difficulty": null
      },
      {
        "id": "c8fcebeb9800834d6191e18953175cf3c578614eec500dcf804deefc1cd72fb6",
        "domain": "numina",
        "question": "456. Every evening, Uncle Chernomor appoints 9 or 10 out of 33 bogatyrs for duty, at his discretion. What is the smallest number of days after which it can happen that each of the bogatyrs has been on duty the same number of times?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 25633,
        "difficulty": null
      },
      {
        "id": "33f4998193d857420f5a07992765db2c8498a2e0a3a42cef6c9b0a6890e8825d",
        "domain": "numina",
        "question": "14. A regular 201-sided polygon is inscribed inside a circle of center $C$. Triangles are drawn by connecting any three of the 201 vertices of the polygon. How many of these triangles have the point $C$ lying inside the triangle?\n14. 一個正 201 邊形內接於一個以 $C$ 為圓心的圓形內。把多邊形 201 個頂點中的任意三個頂點連起, 可以組成不同的三角形。有多少個這樣的三角形, 將 $C$ 點包含在其內部?\n(2 分)",
        "answer": "338350",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 153461,
        "difficulty": null
      },
      {
        "id": "3e732f074fc254978b127bfed040a4a71fba431282e21aa9844c0526b1196d7a",
        "domain": "numina",
        "question": "# 5. Option 1\n\nUnit cubes were used to build a large cube. Two cubes will be called adjacent if they touch by faces. Thus, one cube can have up to 6 neighbors. It is known that the number of cubes that have exactly 4 neighbors is 132. Find the number of cubes that have exactly 5 neighbors.",
        "answer": "726",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18228,
        "difficulty": null
      },
      {
        "id": "0e9daceb5c38b0dd3ac24f4e82b418cb3efe615ba3484b84d0d3713419005542",
        "domain": "numina",
        "question": "18. How many digits does the number $2^{100}$ have?",
        "answer": "31",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 34749,
        "difficulty": null
      },
      {
        "id": "1fcfab94c4a3a39e6b58bc8676b8c14d7638aa671a9f5cd8d180ed090ac39398",
        "domain": "numina",
        "question": "5. There is a paper square $7 \\times 7$, all cells of which are white. What is the minimum number of cells that need to be painted black so that it is impossible to cut out a rectangle from it with at least 10 cells, all of which are white?",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17802,
        "difficulty": null
      },
      {
        "id": "b315963a09ae364e5c9d1e554940e6b0924b25904528cdda5867ca4a9db3e5e1",
        "domain": "numina",
        "question": "2. Write down and read all seven-digit numbers, the sum of the digits in each of which is equal to 2. How many such numbers are there?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 39272,
        "difficulty": null
      },
      {
        "id": "7a2424d25773be7b2ef89d5458e37269cd78db47cd4164b5690641a617dd6b15",
        "domain": "numina",
        "question": "Question 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$.\n\n保留源文本的换行和格式，直接输出翻译结果。 \n\n(注意：最后一句是解释说明，不是原文的一部分，因此在最终输出时应去掉。)\n\nQuestion 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$.",
        "answer": "54",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 122320,
        "difficulty": null
      },
      {
        "id": "48f515e2a5ac5ff8deefbd22daaba30c3308f70f1e3fd051910c3d4551aef7bf",
        "domain": "numina",
        "question": "5. Given that the diameter $A B$ of circle $O$ divides the circle into upper and lower semicircles, and points $C, D$ are on the upper and lower semicircles respectively (neither coinciding with points $A, B$), if $A C=2, A D=1$, then $\\overrightarrow{A B} \\cdot \\overrightarrow{D C}=$ $\\qquad$ .",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 126466,
        "difficulty": null
      },
      {
        "id": "2b39415cbea01e3c5f497c0eb11dc1eb2ee60120dd4d929bd1e8ea999bf521dc",
        "domain": "numina",
        "question": "## Task A-2.6.\n\nIn a dining room measuring $6 \\mathrm{~m} \\times 6 \\mathrm{~m}$, one corner is occupied by a rectangular card table with dimensions $2 \\mathrm{~m} \\times$ $1.5 \\mathrm{~m}$. What is the radius of the largest circular rug that can be spread on the dining room floor?",
        "answer": "2.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88818,
        "difficulty": null
      },
      {
        "id": "5f8a5ccdc8123a57d6900e28cbe4114ae0ec053eae638ef3a869799f9f3c2de9",
        "domain": "numina",
        "question": "22. The Rainforest School, Desert School, and Tundra School in Zootopia are organizing a Spring Festival Gala, with a total of 15 performances. If each school must perform at least 3 programs, then the number of different ways the three schools can perform is $\\qquad$ kinds.",
        "answer": "28",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 111555,
        "difficulty": null
      },
      {
        "id": "08411f3e04789758325023e44ed25f8b6446c24beaad840fc79868d35d82172b",
        "domain": "numina",
        "question": "# Problem 5. (based on materials from the Ural Tournaments)\n\nIn triangle $ABC$, angle $B$ is $80^{\\circ}$. On side $BC$, point $D$ is marked such that $AB = AD = CD$. On side $AB$, point $F$ is marked such that $AF = BD$. On segment $AC$, point $E$ is marked such that $AB = AE$. Find angle $AEF$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_19300bfa006950daee48g-3.jpg?height=1431&width=1697&top_left_y=344&top_left_x=192)",
        "answer": "20",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 20317,
        "difficulty": null
      },
      {
        "id": "976e2e53361c9b375411cfc7f63b8dad862c2f73c6c5eaf41f7ad5e92f940035",
        "domain": "numina",
        "question": "5. Arrange 3 identical black balls and 3 identical white balls in a row from left to right. If, starting from any position (including this position) and counting to the left, the number of black balls is always not less than the number of white balls, such an arrangement is called a \"valid arrangement.\" The probability of a \"valid arrangement\" occurring is $\\qquad$ .",
        "answer": "\\frac{1}{4}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 143852,
        "difficulty": null
      },
      {
        "id": "9acb2054a4cb3fa2818badb33f15fe9575b19c831c6bbfb7f653ba7981d69c8f",
        "domain": "numina",
        "question": "Example 9 The function $f(x, y)$ satisfies for all non-negative integers: (1) $f(0, y)=y+1$; (2) $f(x+1,0)=$ $f(x, 1)$; (3) $f(x+1, y+1)=f(x, f(x+1, y))$. Determine $f(4,1981)$.",
        "answer": "2^{2^{1984}-3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99692,
        "difficulty": null
      },
      {
        "id": "51fdea43c3359a3bed66dd36bce8c3bb5cb5eab4170586d43ca1dbdcef9a881d",
        "domain": "numina",
        "question": "13.063. In the club's auditorium, there were 320 seats arranged in identical rows. After the number of seats in each row was increased by 4 and one more row was added, the auditorium had 420 seats. How many rows are there in the auditorium now?",
        "answer": "21",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51564,
        "difficulty": null
      },
      {
        "id": "adefab36b545c33541263e3cdb8039cc513d7d9a079b72ba5659299e26051eac",
        "domain": "numina",
        "question": "## Task A-1.2.\n\nGiven is a right triangle $A B C$ with a right angle at vertex $C$ and side lengths $|A B|=26$, $|B C|=24$. A semicircle is inscribed in it with the diameter on side $\\overline{B C}$ which contains point $C$. The semicircle touches side $\\overline{A B}$. What is the radius of the inscribed semicircle?",
        "answer": "12",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 87881,
        "difficulty": null
      },
      {
        "id": "2325a78c7405a4a32ebabf038096dd43bd8f9828aebd52712768f7fc3973eeeb",
        "domain": "numina",
        "question": "3. From the 10 numbers $0,1,2,3,4,5,6,7,8,9$, take out 3 numbers so that their sum is an even number not less than 10. The number of different ways to do this is $\\qquad$.",
        "answer": "51",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125808,
        "difficulty": null
      },
      {
        "id": "07876757f7a422a15d3cb410056c20f90c4ab24fe3f51762c984d9ce80434149",
        "domain": "numina",
        "question": "Um decorador distribuirá flores em oito pontos ao redor de um arranjo quadrado de flores, como indicado na figura abaixo. Ele quer fazer isso de modo tal que, em cada lado do arranjo, as pessoas vejam sempre a mesma quantidade de flores. No exemplo abaixo, temos o total de 11 flores e em cada um dos 4 lados do quadrado são vistas exatamente 4 delas.\n\na) Qual o número máximo de flores que podem ser usadas, considerando que em cada lado do quadrado devem ser vistas exatamente 9 flores?\n\nb) Qual o número mínimo de flores que podem ser usadas, considerando que em cada lado do quadrado devem ser vistas exatamente 12 flores?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-19.jpg?height=519&width=514&top_left_y=954&top_left_x=859)\n\n#",
        "answer": "24",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 73950,
        "difficulty": null
      },
      {
        "id": "927336758045ea7f03cd5f9f91e2f3c7784a5b28aadafc4e26f31d249a81db8b",
        "domain": "numina",
        "question": "Problem 11.3. (15 points) Natural numbers $a, b, c$ are such that $1 \\leqslant a<b<c \\leqslant 3000$. Find the greatest possible value of the quantity\n\n$$\n\\text { GCD }(a, b)+\\text { GCD }(b, c)+\\text { GCD }(c, a)\n$$",
        "answer": "3000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 20136,
        "difficulty": null
      },
      {
        "id": "fc28c937e8cea95e83201c64cc3bbf213e43e26f895d26c1a427bb8bac52bf70",
        "domain": "numina",
        "question": "Problem 8.6. For quadrilateral $ABCD$, it is known that $AB=BD, \\angle ABD=\\angle DBC, \\angle BCD=90^{\\circ}$. A point $E$ is marked on segment $BC$ such that $AD=DE$. What is the length of segment $BD$, if it is known that $BE=7, EC=5$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_e7f9c87b6a37ffba3564g-30.jpg?height=476&width=517&top_left_y=91&top_left_x=468)",
        "answer": "17",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15581,
        "difficulty": null
      },
      {
        "id": "b9218f41085f9cce0f6e868c6ed421481a1833b0fc10bcd07a8943b91d158080",
        "domain": "numina",
        "question": "7. In a chess tournament, 8 players participated. They scored $7, 6, 4, 4, 3, 2, 1, 5$ and 0.5 points respectively. How many points did the players who took the first four places lose in matches against the others?",
        "answer": "1",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 35691,
        "difficulty": null
      },
      {
        "id": "77290fb017328693903ed1a11c9c9f9bea9228fa9c546e2460a039dd93fe5b14",
        "domain": "numina",
        "question": "Problem 3. In $1000 \\mathrm{~kg}$ of fresh berries, there is 99% water. During transportation, a certain amount of water evaporated, so the weight of the berries after the water evaporation is $500 \\mathrm{~kg}$. What percentage of water do the berries now contain?",
        "answer": "98",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188726,
        "difficulty": null
      },
      {
        "id": "9b4746d442ec01655d267c7c96defc2ba7c766417e4f7e5b99b9921cbf49817a",
        "domain": "numina",
        "question": "## Task B-3.1.\n\nSolve the equation $5 \\cdot 0.16^{x}+8 \\cdot 0.4^{x}=4$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 87266,
        "difficulty": null
      },
      {
        "id": "dafa8dc1fb4965fd91aa0f991aee905ac9144ca04d8b3e28ee25e178ad4b775e",
        "domain": "numina",
        "question": "3. (CAN) Find the minimum value of  $$ \\max (a+b+c, b+c+d, c+d+e, d+e+f, e+f+g) $$  subject to the constraints (i) $a, b, c, d, e, f, g \\geq 0$, (ii) $a+b+c+d+e+f+g=1$.",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 196848,
        "difficulty": null
      },
      {
        "id": "f00f4d0d6cf0706e78f835c1e29d22be83aea130a470f94cb5561679f066f727",
        "domain": "numina",
        "question": "Determine the integer equal to\n\n$$\n2020+1919+1818+1717+1616+1515+1414+1313+1212+1111+1010 .\n$$",
        "answer": "16665",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 194090,
        "difficulty": null
      },
      {
        "id": "b76251f5d9aca64b2f3fb7e6e5b1749c721aa0eed29d9c7492d175c74913dfa1",
        "domain": "numina",
        "question": "20 Anna randomly picked five integers from the following list\n$$\n53,62,66,68,71,82,89\n$$\nand discover that the average value of the five integers she picked is still an integer. If two of the integers she picked were 62 and 89 , find the sum of the remaining three integers.",
        "answer": "219",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 160340,
        "difficulty": null
      },
      {
        "id": "aa289a6ad0e899f73221ceccd6ec682eb5e38ed13b750980f6697ce0b3e6cd09",
        "domain": "numina",
        "question": "4. Petya wrote down $n$ consecutive two-digit numbers $(n \\geqslant 2)$ in a row on the board, the first of which does not contain the digit 4, and the last one does not contain the digit 7. Vasya thought that this was the decimal representation of a natural number $x$, and he factored $x$ into prime factors. It turned out that there were only two of them and they differ by 4. What is written on the board?",
        "answer": "2021",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 7749,
        "difficulty": null
      },
      {
        "id": "194815c0183193e4745d76b28682a41531b1d1a623853c8301c8aaeb236ac90a",
        "domain": "numina",
        "question": "9. In our country, individual income tax on wages and salaries is calculated in segments. Total income not exceeding 800 yuan is exempt from individual income tax on wages and salaries; the portion exceeding 800 yuan is subject to taxation. Let the taxable amount for the entire month be $x$, where $x=$ total monthly income - 800 yuan. The tax rates are as follows:\n\\begin{tabular}{|c|c|c|}\n\\hline Level & Monthly taxable amount $x$ & Tax \\\\\n\\hline 1 & Up to 500 yuan \\\\\n\\hline 2 & Over 500 yuan to 2000 yuan & $5 \\%$ \\\\\n\\hline 3 & Over 2000 yuan to 5000 yuan & $10 \\%$ \\\\\n\\hline$\\vdots$ & & $15 \\%$ \\\\\n\\hline 9 & Over 100000 yuan & $45 \\%$ \\\\\n\\hline\n\\end{tabular}\n(1) If the tax amount is $f(x)$, try to express the calculation formula for the tax amount $f(x)$ of levels 1-3 using a piecewise function.\n(2) A person's total monthly income in March 2000 was 3000 yuan. Calculate how much personal income tax this person should pay in March.",
        "answer": "205",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 92280,
        "difficulty": null
      },
      {
        "id": "de4fa1cdb5712600f07d1163c12a67c27e4667c0f2fa83d991fc2432b53b0c7c",
        "domain": "numina",
        "question": "246. $\\int 8 x^{3} d x$",
        "answer": "160",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32166,
        "difficulty": null
      },
      {
        "id": "0019c24135f44f8ef65bd50b127ca0b433f3b7f2ebd893b2f1211d4e8554bbf4",
        "domain": "numina",
        "question": "10.3. Point $F$ is the midpoint of side $B C$ of square $A B C D$. A perpendicular $A E$ is drawn to segment $D F$. Find the angle $C E F$.\n\n---\n\nTranslation provided as requested, maintaining the original format and line breaks.",
        "answer": "45",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15683,
        "difficulty": null
      },
      {
        "id": "a8eba6d5d257c1a140bcee98cae894db16dcd29a92f99954aa12999a13d227de",
        "domain": "numina",
        "question": "[Arithmetic. Mental arithmetic, etc.]\n\nThe tower clock strikes three times in 12 seconds. How long will it take for it to strike six times?\n\n#",
        "answer": "30",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 69085,
        "difficulty": null
      },
      {
        "id": "699c2565bc140dc4df5609cd714ae430b30356ed342a70b413dec787d239c258",
        "domain": "numina",
        "question": "4. Given $(x+3)(x-1)^{4}=a_{0}+a_{1} x+a_{2} x^{2}+a_{3} x^{3}+a_{4} x^{4}+a_{5} x^{5}$, then $a_{3}+a_{4}+a_{5}=$",
        "answer": "-6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99080,
        "difficulty": null
      },
      {
        "id": "ca50d0218d88f6eb6edb030613161e0275fc5283d344a4d5dff5a1e1bc24881c",
        "domain": "numina",
        "question": "18. The number of real solutions to the equation $\\sin \\left(x-\\frac{\\pi}{4}\\right)=\\frac{1}{4} x$ is",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 109022,
        "difficulty": null
      },
      {
        "id": "15675c0da8e019138276e1e95401793e0484d46f2b3770cbbb2013cc59b47393",
        "domain": "numina",
        "question": "7. In the Magic Land, there is a magic stone that grows uniformly upwards. To prevent it from piercing the sky, the Elders of the Immortal World decided to send plant warriors to consume the magic stone and suppress its growth. Each plant warrior consumes the same amount every day. If 14 plant warriors are dispatched, the magic stone will pierce the sky after 16 days; if 15 plant warriors are dispatched, the magic stone will pierce the sky after 24 days. At least $\\qquad$ plant warriors need to be dispatched to ensure the sky is not pierced.",
        "answer": "17",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 138273,
        "difficulty": null
      },
      {
        "id": "9757fe78cf779ea9a60e638910a948ead0245a3593504686f1953d8490991e62",
        "domain": "numina",
        "question": "Problem 3. Vasya wants to find all integers $a$ such that the expression $10 n^{3}+3 n^{5}+7 a n$ is divisible by 15 for all integers $n$. What remainders can the number $a$ give when divided by 15? Provide all possible answers or prove that there are no such integers $a$.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9232,
        "difficulty": null
      },
      {
        "id": "2bc3bfcf45bcb370b5b8769a88e81337fa618427b9bd3e80a6913d3efe1401d5",
        "domain": "numina",
        "question": "## Task 5 - 311235\n\nDetermine whether it is possible to select fifty different natural numbers from 1 to 100 such that their sum is 2525 and no two of them add up to 101.",
        "answer": "2525",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 186228,
        "difficulty": null
      },
      {
        "id": "b56a1be24ce0b1b13950840907c2ece669e816035baa88d4619a6273e673da89",
        "domain": "numina",
        "question": "3. The product of three natural numbers is 13600. Calculate the product of the first and second number if the product of the first and third number is 544, and the product of the second and third is 425.",
        "answer": "800",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 86532,
        "difficulty": null
      },
      {
        "id": "a10957461097c3fd1af37d920a8db7f1f0fb412a55488e9efdac0248e63e9571",
        "domain": "numina",
        "question": "Juarez used the digits 1, 2, 3, 4, and 5 to write the five-digit number $\\overline{a b c d e}$. Without revealing what this number is, he told Luciana that:\n\n$\\cdot$ the number $\\overline{a b c}$ is divisible by 4;\n\n- the number $\\overline{b c d}$ is divisible by 5;\n\n$\\cdot$ the number $\\overline{c d e}$ is divisible by 3.\n\nThen, Luciana told Juarez that it is possible to figure out what the number $\\overline{a b c d e}$ is. Show that Luciana is correct, that is, find the number $\\overline{a b c d e}$.",
        "answer": "12453",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 73241,
        "difficulty": null
      },
      {
        "id": "345f27e6f60fad88e5b1ad1e28ebdf2c32ea4dcc35dade4e5601171589bb5254",
        "domain": "numina",
        "question": "## 18. Expected Lifespan\n\nAn ethnographer found that in a certain primitive tribe he studied, the distribution of the lifespan of the tribe members could be described as follows: $25 \\%$ of them only lived up to 40 years, $50 \\%$ died at 50 years, and $25 \\%$ - at 60 years. He then randomly selected two individuals to study in more detail. What is the expected lifespan of the one of the two randomly selected individuals who will live longer?",
        "answer": "53",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 41996,
        "difficulty": null
      },
      {
        "id": "167580c31df7dc093d42ef215829afb6577ce6bd342d361a9bc61a1011a1e227",
        "domain": "numina",
        "question": "4. In triangle $\\triangle A B C$, the base $A B$ of which lies on the x-axis, altitudes $A M, B N$ and $C K$ are drawn. Find the length of the base $A B$, if the coordinates of points $M(2,2)$ and $N(4,4)$ are known.",
        "answer": "4\\sqrt{5}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 6309,
        "difficulty": null
      },
      {
        "id": "f5628a9cd551d769ce76659aa79c486a07e7dbc9c0f0f006c3ccab95a4a714d5",
        "domain": "numina",
        "question": "13. (6 points) Two classes in Grade 4 have a total of 72 students, among whom there are 35 girls. Class 4(1) has 36 students, and Class 4(2) has 19 boys. Therefore, Class 4(1) has $\\qquad$ girls.",
        "answer": "18",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 123000,
        "difficulty": null
      },
      {
        "id": "20b92d919bb4f37e5b0afd437b04024e22bd5efb4b64929c2865a6a2fe70fa53",
        "domain": "numina",
        "question": "Miscellaneous Problem 1 The crossed ladder problem. In an alley a ladder length 3 is propped up against one wall with its foot at the bottom of the other wall. It crosses another ladder length 2 leaning the other way a distance 1 from the ground. How wide is the alley? [In other words we are given a quadrilateral ABCD. Angles ADC and DCB are right angles. AC = 2, BD = 3 and AC, BD intersect at X a distance 1 from CD. Find CD.]",
        "answer": "1.2311857",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 150199,
        "difficulty": null
      },
      {
        "id": "82c2b459f77e644f581b60feeb58e371f35ad442aab1fc3e5c30f254f08dc79a",
        "domain": "numina",
        "question": "5. Place eight balls with a radius of 1 in two layers inside a cylinder, such that each ball is tangent to its four adjacent balls, and is also tangent to one base and the side of the cylinder. Then the height of this cylinder is $\\qquad$ .",
        "answer": "\\sqrt[4]{8}+2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 147167,
        "difficulty": null
      },
      {
        "id": "1f77fbdc96b02889fbb4986dd4a8f59ab69cfe98fdd40d81428a3ff66094f6c4",
        "domain": "numina",
        "question": "3 [Divisibility of numbers. General properties]\n\nIt is known that $x=2 a^{5}=5 b^{2}>0$, where numbers $a$ and $b$ are integers. What is the smallest possible value of $x$?",
        "answer": "200000",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 69715,
        "difficulty": null
      },
      {
        "id": "a3c9120d59d5511d6b4163d1241cbd78c4e251387c851af40ee373590d2e75ca",
        "domain": "numina",
        "question": "Problem 9.4. From left to right, intersecting squares with sides $12, 9, 7, 3$ are depicted respectively. By how much is the sum of the black areas greater than the sum of the gray areas?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_a7569897979ba1659de3g-34.jpg?height=686&width=872&top_left_y=927&top_left_x=289)",
        "answer": "103",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 16195,
        "difficulty": null
      },
      {
        "id": "48e544f25b46269734f296666b718f54b5de215a0cc7a940559cf04f01a5ef1e",
        "domain": "numina",
        "question": "Example: Sets $A, B, C$ (not necessarily distinct) satisfy $A \\cup B \\cup C=\\{1,2,3, \\cdots, 11\\}$. Find the number of solutions of the ordered triples $(A, B, C)$ that meet the condition.",
        "answer": "7^{11}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125493,
        "difficulty": null
      },
      {
        "id": "73c9b293529461b5c729302cc97247d4728abf073331605c6702f752846ab1f6",
        "domain": "numina",
        "question": "113. The base of the pyramid $A B C E H$ is a convex quadrilateral $A B C E$, which is divided into two equal-area triangles by the diagonal $B E$. The length of the edge $A B$ is 1, the lengths of the edges $B C$ and $C E$ are equal. The sum of the lengths of the edges $A H$ and $E H$ is $\\sqrt{2}$. The volume of the pyramid is $1 / 6$. Find the radius of the sphere with the largest volume among all spheres that can fit inside the pyramid.",
        "answer": "\\frac{3-\\sqrt{5}}{4}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 29822,
        "difficulty": null
      },
      {
        "id": "f528a96cda16f4bdb59090ade00f62f592050712e9a2d0dcba5d5139a1b4f427",
        "domain": "numina",
        "question": "# Problem 7. (4 points)\n\nIn a $7 \\times 7$ table, some cells are black, and the rest are white. In each white cell, the total number of black cells on the same row or column is written; nothing is written in the black cells. What is the maximum value that the sum of the numbers in the entire table can take?\n\n#",
        "answer": "168",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 20447,
        "difficulty": null
      },
      {
        "id": "0fd09ebbd226772292fe218ddbdbc716d7ebdda4913d70af8ea04bddb8c9252b",
        "domain": "numina",
        "question": "1. Andrei, Boris, and Valentin participated in a 1 km race. (We assume that each of them ran at a constant speed). Andrei was 50 m ahead of Boris at the finish line. And Boris was 40 m ahead of Valentin at the finish line. What was the distance between Andrei and Valentin at the moment Andrei finished?",
        "answer": "88",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 4036,
        "difficulty": null
      },
      {
        "id": "b18481738c984ce4a819f55593b655c385cf07e621edce76e4df4ab8ada1da0f",
        "domain": "numina",
        "question": "3. The villages of Arkadino, Borisovo, and Vadimovo are connected by straight roads. A square field adjoins the road between Arkadino and Borisovo, one side of which completely coincides with this road. A rectangular field adjoins the road between Borisovo and Vadimovo, one side of which completely coincides with this road, and the second side is 4 times longer. $\\mathrm{K}$ road between Arkadino and Vadimovo adjoins a rectangular forest, one side of which completely coincides with this road, and the second side is 12 km. The area of the forest is 45 sq. km greater than the sum of the areas of the fields. Find the total area of the forest and fields in sq. km.\n\n$\\{135\\}$",
        "answer": "135",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 11153,
        "difficulty": null
      },
      {
        "id": "c0f4e77f8372f75e09d95befe10ffdcb01b55b31e43314824737ac19cfcd8de5",
        "domain": "numina",
        "question": "2+ [ Classical combinatorics (other) $]\n\nIn a bag, there are 70 balls that differ only in color: 20 red, 20 blue, 20 yellow, and the rest are black and white.\n\nWhat is the smallest number of balls that need to be drawn from the bag, without seeing them, to ensure that among them there are at least 10 balls of the same color?\n\n#",
        "answer": "38",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 57003,
        "difficulty": null
      },
      {
        "id": "9ca92f73b6dc3ea3c3e48441bad15780f91343dc3e6bdc0e09177e6225ac18d6",
        "domain": "numina",
        "question": "13.262. It is known that a freely falling body travels 4.9 m in the first second, and in each subsequent second, it travels 9.8 m more than in the previous one. If two bodies start falling from the same height, one 5 seconds after the other, then after what time will they be 220.5 m apart?",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49349,
        "difficulty": null
      },
      {
        "id": "2fa0114b046b70fd2bf87d0f7b1221bf2d67dc2ddc223ea4b5acbef9a9e8c92e",
        "domain": "numina",
        "question": "As shown in the figure, find the sum of the degrees of the 12 angles marked with numbers.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "720",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 103193,
        "difficulty": null
      },
      {
        "id": "1fbc41a73b518f4d20821ed8d1971065060e43ef45cf1ead78607f14764865ac",
        "domain": "numina",
        "question": "Task 1. 600 kg of mushrooms with a moisture content of $98 \\%$ were collected. After drying, the moisture content was reduced to $96 \\%$. What is the mass of the mushrooms after drying?",
        "answer": "300",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 188764,
        "difficulty": null
      },
      {
        "id": "d5aa5ddcb790ef5dde4b256cfcd63df3b30485ff0916c169389d91122a6f730b",
        "domain": "numina",
        "question": "Let's determine all natural numbers $a, b, c, d$ for which $a b=c+d$, and $a+b=c d$ hold.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 168769,
        "difficulty": null
      },
      {
        "id": "513e25d7af49ee6edfbf696ed51561be791e67db6d6a75245c3834c421efbdd9",
        "domain": "numina",
        "question": "Is there such an $n$, and if there is, how large is it, for which the sum of the first $n$ natural numbers is a three-digit number with equal digits?",
        "answer": "36",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 174226,
        "difficulty": null
      },
      {
        "id": "208fba4e7fa4d17f5001fddfd166f37f6e5723c78f7a402e05ad7ea8e4acadbd",
        "domain": "numina",
        "question": "1. Immediately after harvesting, the water content in 1 ton of tomatoes was $99 \\%$. By the time of sale, the water share had decreased by $4 \\%$ (after drying). What did the total weight of the tomatoes become?",
        "answer": "200",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 14513,
        "difficulty": null
      },
      {
        "id": "b76b7499c97de960434b0c492505ba60e79cd0ff2bbeaf77c4bfa7df7b60bc00",
        "domain": "numina",
        "question": "SI. 2 In Figure $1, A B=A C$ and $A B / / E D$.\nIf $\\angle A B C=P^{\\circ}$ and $\\angle A D E=Q^{\\circ}$, find the value of $Q$.",
        "answer": "120",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 153286,
        "difficulty": null
      },
      {
        "id": "0e582028d3eff274a66641eb4f33e661efa72eb9e9ee6f26ea0241250e1db063",
        "domain": "numina",
        "question": "8. Let $x, y$ be nonnegative integers such that $x+2 y$ is a multiple of $5, x+y$ is a multiple of 3 and $2 x+y \\geq 99$. Find the minimum possible value of $7 x+5 y$.\n設 $x 、 y$ 少非負整數, 使得 $x+2 y$ 爲 5 的倍數、 $x+y$ 絧 3 的倍數, 且 $2 x+y \\geq 99$ 。求 $7 x+5 y$ 的最小可能値。",
        "answer": "366",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153558,
        "difficulty": null
      },
      {
        "id": "6844fa3c96c38ebd4c8ecee57686f684a2c1082dc387bc24c76feb2da7b2947a",
        "domain": "numina",
        "question": "A square is divided into 16 equal squares, obtaining the set of 25 different vertices. What is the least number of vertices one must remove from this set, so that no 4 points of the remaining set are the vertices of any square with sides parallel to the sides of the initial square?\n\nRemark. The proposed solution to this problem claimed that it is enough to remove 7 vertices but the example to demonstrate this appeared to be incorrect. Below we show that removing 6 vertices is not sufficient but removing 8 vertices is. It seems that removing 7 vertices is not sufficient but we currently know no potential way to prove this, apart from a tedious case study.",
        "answer": "8",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 72932,
        "difficulty": null
      },
      {
        "id": "820820d0b795dce4b46ab5206f13709744ec36c978b5fc36da071b1410a5e62f",
        "domain": "numina",
        "question": "9.4. In a square table of size $100 \\times 100$, some cells are shaded. Each shaded cell is the only shaded cell either in its column or in its row. What is the maximum number of cells that can be shaded?",
        "answer": "198",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 13130,
        "difficulty": null
      },
      {
        "id": "32b9ffd67beaf4e7c433d09a2084ee2aef495d9b2cf1b14621235e366dd64497",
        "domain": "numina",
        "question": "3. A rectangle $A D E C$ is described around a right triangle $A B C$ with legs $A B=5$ and $B C=6$, as shown in the figure. What is the area of $A D E C$?",
        "answer": "30",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 5332,
        "difficulty": null
      },
      {
        "id": "81efd5cdfc11a9cb06bc2bc7a95f5e2ea9b734f76602c73d36b5f5a66bc7ab5e",
        "domain": "numina",
        "question": "4. The Petrovs family decided to buy an apartment with a mortgage. The total cost of the apartment was 13,000,000 rubles, and the family did not have enough funds, so they decided to take a loan of 8,000,000 rubles. One of the bank's conditions for issuing the loan was the purchase of a comprehensive mortgage insurance policy for the first year of the loan, including insurance of the acquired apartment against risks of loss and damage (property insurance), title insurance (loss of ownership rights to the apartment), and life and health insurance of the borrowers. Maria Sergeyevna Petrova and Vasily Dmitrievich Petrov became co-borrowers in the ratio of 40% and 60% of the loan amount, respectively. Calculate the family's expenses on insurance for the year if the bank's requirement is formulated as follows: \"The insurance amount for the risks of loss, damage, and loss of ownership for the first year of insurance should be no less than the loan amount at the time of issuance, increased by the interest rate, and the insurance amount for the life and health insurance of each co-borrower, calculated in the above-mentioned manner, is divided proportionally according to each of their contributions to the loan.\" (20 points)\n\nAppendix: the interest rate on the loan is 9.5%, the tariff for property insurance is 0.09%, life and health insurance (women) is 0.17%, life and health insurance (men) is 0.19%, and title insurance is 0.27%.",
        "answer": "47481.20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 11582,
        "difficulty": null
      },
      {
        "id": "1967bdc516e18e54ce31f4a196e6ede4fe458b0666c12ee086348cf2022b821e",
        "domain": "numina",
        "question": "13.222. The distance between stations $A$ and $B$ is 103 km. A train left $A$ for $B$ and, after traveling a certain distance, was delayed, and therefore the remaining distance to $B$ was traveled at a speed 4 km/h greater than the initial speed. Find the original speed of the train, given that the remaining distance to $B$ was 23 km longer than the distance traveled before the delay, and that the time taken to travel the remaining distance after the delay was 15 minutes more than the time taken to travel the distance before the delay.",
        "answer": "80",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49310,
        "difficulty": null
      },
      {
        "id": "eecf2752a979261eba4bfda77544a4e4de937ad49fef74c8821d77714f2a66b3",
        "domain": "numina",
        "question": "In the plane, let there be a parabola. Let its vertex be $O$ and an arbitrary point on it be $M$. We draw a perpendicular to the line $OM$ at point $O$, and draw a line parallel to the axis of the parabola through point $M$. Determine the geometric locus of the intersection point of the two lines as $M$ runs along the parabola.",
        "answer": "-4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 174929,
        "difficulty": null
      },
      {
        "id": "07555a5b2b8ccebd05ba1deef86982861dd60d1a3c01450f512fdbb626e014a9",
        "domain": "numina",
        "question": "12. The smallest positive period of the function $f(x)=\\cos ^{3} x-\\sin ^{2} x$ is",
        "answer": "2\\pi",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 120797,
        "difficulty": null
      },
      {
        "id": "66ff3cf7118294dfaff8b96a571c7678e9eec62baa84298ef08545f0f59bba46",
        "domain": "numina",
        "question": "1. In triangle $\\mathrm{ABC}$, angle $\\mathrm{C}$ is a right angle, and $\\mathrm{CD}$ is the altitude. Find the length of the radius of the circle inscribed in triangle $\\mathrm{ABC}$, if the lengths of the radii of the circles inscribed in triangles $\\mathrm{ACD}$ and $\\mathrm{BCD}$ are 6 and 8, respectively.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 21945,
        "difficulty": null
      },
      {
        "id": "9be327bf223ea53bcc6a637d9f1f9d0e73fcdaa11beeaf612a11b8eb7d22b597",
        "domain": "numina",
        "question": "Problem 6.5. In the park, paths are laid out as shown in the figure. Two workers started to asphalt them, starting simultaneously from point $A$. They lay asphalt at constant speeds: the first on the section $A-B-C$, the second on the section $A-D-E-F-C$. In the end, they finished the work simultaneously, spending 9 hours on it. It is known that the second works 1.2 times faster than the first. How many minutes did the second spend laying asphalt on the section $D E$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_899cf5197845501d962eg-16.jpg?height=259&width=299&top_left_y=1340&top_left_x=577)",
        "answer": "45",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15787,
        "difficulty": null
      },
      {
        "id": "f32b7b1d8fc28792065af48151d39cb43410339087829a9dac0468505a498799",
        "domain": "numina",
        "question": "14. Place the numbers $1,2,3,4,5$ in a circle. We call a placement method a \"central ring\" placement if for some number $n$ in the range 1 to 15, it is impossible to select several consecutive numbers on the circle such that their sum is $n$. If two placement methods are the same after rotation or reflection, they are considered the same method. How many \"central ring\" placement methods are there in total?\n(Explanation: If placed as shown in the figure below, then $1-5$ can be taken as a single number, $6=5+1,7=3+4,8=5+1+2$, $9=2+3+4,10=1+2+3+4,11=5+1+2+3,12=4+5+1+2,13=3+4+5+1,14=2+3+4+5,15=1+2+3+4+5$, so all numbers from 1 to 15 are obtained, and this is not a \"central ring\" placement method.)",
        "answer": "2",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 93268,
        "difficulty": null
      },
      {
        "id": "0977ad283baeebe698b26e130af3e04706cd3bba0e3c2750a28fced587264435",
        "domain": "numina",
        "question": "5. In an $8 \\times 8$ table, natural numbers are arranged. The numbers in cells symmetric with respect to both diagonals of the table are equal. It is known that the sum of all numbers in the table is 1000, and the sum of the numbers on the diagonals is 200. For what smallest number $M$ can we assert that the sum of the numbers in each row does not exceed $M$?",
        "answer": "288",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 8083,
        "difficulty": null
      },
      {
        "id": "3a07e450e436e46b00967861923e7666505e141774ded17e382dd2a85054de97",
        "domain": "numina",
        "question": "Three. (50 points) Given $n \\in \\mathbf{N}, n>4$, the set $A=$ $\\{1,2, \\cdots, n\\}$. Let the positive integer $m$ and the sets $A_{1}, A_{2}, \\cdots$, $A_{m}$ have the following properties:\n(1) $\\bigcup_{i=1}^{m} A_{i}=A$;\n(2) $\\left|A_{i}\\right|=4(i=1,2, \\cdots, m)$;\n(3) $X_{1}, X_{2}, \\cdots, X_{\\mathrm{C}_{n}^{2}}$ are all the two-element subsets of $A$, for any $X_{k}\\left(k=1,2, \\cdots, \\mathrm{C}_{n}^{2}\\right)$, there exists a unique $j_{k} \\in\\{1,2, \\cdots, m\\}$, such that $X_{k} \\subseteq A_{j_{k}}$.\nFind the minimum value of $n$.",
        "answer": "13",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 125235,
        "difficulty": null
      },
      {
        "id": "c8dea2d6c7c7531247b6c4c686734d622ef11a62436944ac8a31655e67873858",
        "domain": "numina",
        "question": "7.3. Egor borrowed 28 rubles from Nikita, and then returned them in four payments. It turned out that Egor always returned a whole number of rubles, and the amount of each payment increased and was divisible by the previous one. What was the last amount Egor paid?",
        "answer": "18",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 9464,
        "difficulty": null
      },
      {
        "id": "47a8b39eff2761289f5268c50d32d48532844f0d90b8d2989227e88267eed34a",
        "domain": "numina",
        "question": "102. A certain number is written in the duodecimal system; for which divisor $m$ is the following divisibility rule valid: if the sum of the digits of the number is divisible by $m$, then the number is also divisible by $m$?",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 39938,
        "difficulty": null
      },
      {
        "id": "9b8511c451dcd43eb2339a8da336f087b655f8b1c5d4ae32689cadbaf868c157",
        "domain": "numina",
        "question": "G9.2 There are exactly $n$ values of $\\theta$ satisfying the equation $\\left(\\sin ^{2} \\theta-1\\right)\\left(2 \\sin ^{2} \\theta-1\\right)=0$, where $0^{\\circ} \\leq \\theta \\leq 360^{\\circ}$. Find $n$.",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152555,
        "difficulty": null
      },
      {
        "id": "bf2ba78dbff14c352762d51b6ca4cda1a53be72eef546f49b8d5926118e384be",
        "domain": "numina",
        "question": "111) Find the maximum value of the real number $\\lambda$. Such that, as long as point $P$ is inside the acute $\\triangle ABC$, $\\angle PAB = \\angle PBC = \\angle PCA$, the rays $AP$, $BP$, $CP$ intersect the circumcircles of $\\triangle PBC$, $\\triangle PCA$, $\\triangle PAB$ at points $A_1$, $B_1$, $C_1$ respectively, then $S_{\\triangle B C} + S_{\\triangle B_1 C A} + S_{\\triangle C_1 A B} \\geq \\lambda S_{\\triangle A B C}$.",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 118234,
        "difficulty": null
      },
      {
        "id": "a34f6b824ce116cd5b21fc4e752dedb46cf05430124f237a7aa1a13e76ee9604",
        "domain": "numina",
        "question": "Example 1 (2004 National College Entrance Examination for Science) Given that there are 3 weak teams among 8 teams, the 8 teams are divided into groups $A$ and $B$ by drawing lots, with 4 teams in each group.\n(1) The probability that one of the groups $A$ or $B$ has exactly two weak teams;\n(2) The probability that group $A$ has at least two weak teams.",
        "answer": "\\frac{6}{7}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 108652,
        "difficulty": null
      },
      {
        "id": "b7777c9deefbe2efb583a22df0723c0a5bf9a7786e54e87c36c5f5e0901385be",
        "domain": "numina",
        "question": "14. (15 points) As shown in the figure, the radii of circular gears $A$, $B$, and $C$ are 15 cm, 10 cm, and 5 cm, respectively, and they are part of a transmission mechanism. When $A$ rotates uniformly, it drives $B$ to rotate uniformly, which in turn drives $C$ to rotate uniformly. Please answer:\n(1) When $A$ rotates uniformly in a clockwise direction, does $C$ rotate in a clockwise or counterclockwise direction?\n(2) When $A$ completes one full rotation, how many full rotations does $C$ complete?",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 134107,
        "difficulty": null
      },
      {
        "id": "ccaf66433d7f640d5d833eedc13e60a2013d961d4967b9a38ad7b99c00b644be",
        "domain": "numina",
        "question": "5.1. (12 points) The equation $x^{2}+5 x+1=0$ has roots $x_{1}$ and $x_{2}$. Find the value of the expression\n\n$$\n\\left(\\frac{x_{1} \\sqrt{6}}{1+x_{2}}\\right)^{2}+\\left(\\frac{x_{2} \\sqrt{6}}{1+x_{1}}\\right)^{2}\n$$",
        "answer": "220",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6165,
        "difficulty": null
      },
      {
        "id": "84a099bba2bbee4d156e8b63d0c29cea4944468618bd23cf67ad4c68189815e7",
        "domain": "numina",
        "question": "3. The height of an isosceles trapezoid is equal to $h$, and its area is equal to $h^{2}$. Determine the angle between the diagonals of this trapezoid.",
        "answer": "90",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 188119,
        "difficulty": null
      },
      {
        "id": "3cce87f5af3554567006da7f09997398b5189c388aa0a6245f2dbf9141945b3c",
        "domain": "numina",
        "question": "44. Given that the time it takes for a mouse to run 5 steps is the same as the time it takes for a cat to run 4 steps, and the distance a mouse runs in 9 steps is the same as the distance a cat runs in 7 steps. Now, the mouse and the cat are 2 meters apart. The cat starts to chase the mouse, the cat needs to run $\\qquad$ meters to catch the mouse.",
        "answer": "72",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 116542,
        "difficulty": null
      },
      {
        "id": "a840b56cf8a4466dfd39404f6d82c10fe5ad3c2978e3079a6b0ff541635d7af7",
        "domain": "numina",
        "question": "149. Scattered set. In a printing house, a multiplication line of the form $\\overline{a b c} \\cdot \\overline{b c a} \\cdot \\overline{c a b}^{*}$ was being set; but the set scattered, and the digits of the product got mixed up. As a result, the product was printed as 2342355286. It is known that $a>b>c$ and that the units digit of the printed product is the correct digit 6. Restore the true value of the product.",
        "answer": "328245326",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 41582,
        "difficulty": null
      },
      {
        "id": "6c61ab8fdbae034e9d768c8861ab1c7a67eed74971a1aa293a36c7b7f09c89ad",
        "domain": "numina",
        "question": "Question 192: Nine small balls numbered $1, 2, \\ldots, 9$ are randomly placed on nine equally spaced points on a circle, with one ball on each point. Let $S$ be the sum of the absolute differences of the numbers on all adjacent balls on the circle. Find the probability of the arrangement that minimizes $S$ (Note: If one arrangement can be made to coincide with another by rotation or reflection, they are considered the same arrangement).",
        "answer": "\\frac{1}{315}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 98629,
        "difficulty": null
      },
      {
        "id": "0ced655c149aee0f6f8a6cf0871b41b70cd1274a0e7f8973b7e05865b0a4e252",
        "domain": "numina",
        "question": "The surface of a 3 x 3 x 3 Rubik's cube consists of 54 cells. What is the maximum number of cells that can be marked so that the marked cells do not share any vertices?\n\n#",
        "answer": "14",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 53955,
        "difficulty": null
      },
      {
        "id": "032390776f9379e1bc66daad7f12f4d3a02c6b92b9501cc2ff5e719f0e93a1ee",
        "domain": "numina",
        "question": "Condition of the problem\n\nCalculate the limit of the numerical sequence:\n$\\lim _{n \\rightarrow \\infty} \\frac{e^{1 / n}+\\sin \\left(\\frac{n}{n^{2}+1}\\right) \\cdot \\cos n}{1+\\cos \\left(\\frac{1}{n}\\right)}$",
        "answer": "\\frac{1}{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 46502,
        "difficulty": null
      },
      {
        "id": "8f54e806a84131af59525a93d30f5509c60bd72cb7e8f4e55f87b3a8e10bf27b",
        "domain": "numina",
        "question": "Jirka drew a square grid with 25 squares, see the image. Then he wanted to color each square so that squares of the same color do not share any vertex.\n\nHow many colors did Jirka need at least?\n\n(M. Dillingerová)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_adee0188cf1f8dcaeca5g-4.jpg?height=383&width=380&top_left_y=485&top_left_x=838)\n\nHint. Start in one of the corner squares.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 76279,
        "difficulty": null
      },
      {
        "id": "3ffbbb4617ed44c57fc86bc3348844dc2a7a302e62d979629691fc98ceba3d85",
        "domain": "numina",
        "question": "Three distinct numbers are given. The average of the average of the two smaller numbers and the average of the two larger numbers is equal to the average of all three numbers. The average of the smallest and the largest number is 2022.\n\nDetermine the sum of the three given numbers.\n\n(K. Pazourek)\n\nHint. Express one of the numbers in terms of the remaining two.",
        "answer": "6066",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76902,
        "difficulty": null
      },
      {
        "id": "6886ff76e6a7e2005c7248bddf84f0e63d718fd31b3403899a3e45bb3002fa71",
        "domain": "numina",
        "question": "114 The sides of a right triangle are all positive integers. One of the legs is 1575 less than the hypotenuse, and the other leg is less than 1991. Then the length of the hypotenuse of this right triangle is $\\qquad$ .",
        "answer": "1799",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 121416,
        "difficulty": null
      },
      {
        "id": "80d11e695d9f094aaea77166c1423f4eea861c2eb037a62e85baef112ed6695c",
        "domain": "numina",
        "question": "363. The velocity of a material point is given by the formula $v=\\left(4 t^{3}-2 t+1\\right)$ m/s. Find the distance traveled by the point in the first 4 s from the start of the motion.",
        "answer": "244",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 32198,
        "difficulty": null
      },
      {
        "id": "1cf520bc1d09cab9e66ea1131996af299624a5bec79df7adfed200feda10cedc",
        "domain": "numina",
        "question": "5. In a football tournament where each team played against each other once, teams $A$, B, V, G, D, and E participated. Teams received 3 points for a win, 1 point for a draw, and 0 points for a loss. In the end, it turned out that teams $A$, B, V, G, and D each scored 7 points. What is the maximum number of points that team E could have scored?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 6717,
        "difficulty": null
      },
      {
        "id": "eea53a917280736a7846c3abb0bedad4282a268b7388f09496f755f36a74bd73",
        "domain": "numina",
        "question": "B3 In the figure, three squares are inscribed in a right-angled triangle (see figure). Calculate the side $x$ of the third square exactly.\n\n(6 points)\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_c987701c9bb62194911fg-05.jpg?height=371&width=739&top_left_y=1077&top_left_x=1178)",
        "answer": "\\frac{50}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 164131,
        "difficulty": null
      },
      {
        "id": "c7a34b39c5271e548b783f4dd4eea6ce705783d743811e6a96491ce646ee8308",
        "domain": "numina",
        "question": "B2. The expression is $2-\\frac{x-2}{5}+\\frac{1+x}{4}$.\n\nA. Calculate the value of the expression for $x=5$.\n\nB. Determine the value of $x$ such that the value of the expression is 4.\n\nC. Determine the value of $x$ such that the value of the expression is at least 2.\n\nD. Find the smallest natural number $x$ such that the value of the expression is an integer.",
        "answer": "7",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 163532,
        "difficulty": null
      },
      {
        "id": "4f3971da1b3aa029f8cb8d871466e712cd23e03eccaf73d7f8d641fd147af3a2",
        "domain": "numina",
        "question": "Raskinna I.V.\n\nWinnie-the-Poo, Piglet, Rabbit, and Donkey Eeyore emptied a barrel of honey. Piglet ate half of what Winnie-the-Poo ate, Rabbit ate half of what Winnie-the-Poo did not eat, and Donkey Eeyore got only a tenth of the barrel. What part of the barrel did Rabbit get?",
        "answer": "\\frac{3}{10}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 68285,
        "difficulty": null
      },
      {
        "id": "8c5966e44c7040063b1c995a74a68557573917ffe30a20bf6449f0937a1716ea",
        "domain": "numina",
        "question": "$\\frac{\\sin \\alpha \\sin (\\alpha+\\beta)+\\cos \\alpha \\cos (\\alpha+\\beta)}{\\cos \\beta}=?$",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 175536,
        "difficulty": null
      },
      {
        "id": "ca76e5e6c83f9e61f2d9a19fe4dd540d7e9ca298e987857637e1441bf0694a36",
        "domain": "numina",
        "question": "6. (10 points) A class plants trees on Tree Planting Day, divided into 3 groups. The first group plants 5 trees per person, the second group plants 4 trees per person, and the third group plants 3 trees per person. It is known that the number of people in the second group is one-third of the sum of the number of people in the first and third groups, and the number of trees planted by the second group is 72 less than the sum of the number of trees planted by the first and third groups. Then, the class has at least $\\qquad$ people.",
        "answer": "32",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 134827,
        "difficulty": null
      },
      {
        "id": "53fcf26e8c2a61bf5c1f004b939426c3b8449b15784759656d76df71bc9b0a5e",
        "domain": "numina",
        "question": "1. (5 points) Calculate: $3 \\frac{3}{4} \\times 1.3 + 3 \\div 2 \\frac{2}{3}=$",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112863,
        "difficulty": null
      },
      {
        "id": "09b9d6e6c48163748450808ac9fe6753d08fcdcbd946b912aa348ee520e1bea1",
        "domain": "numina",
        "question": "Question 64: Given $a, b \\in [1, \\sqrt{3}]$, find the maximum and minimum values of $\\frac{a^{2}+b^{2}-1}{a b}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 147284,
        "difficulty": null
      },
      {
        "id": "f33c4ba5a2ef52c7842903c71e348856ea21a6ef1bcb0de9aed9a805506f4ac5",
        "domain": "numina",
        "question": "10. In a math competition, there are 5 questions, each with a different natural number score. The smaller the question number, the fewer points the question is worth (for example, the score of question 1 is less than the score of question 2). Xiao Ming answered all the questions correctly, and his total score for the first 2 questions is 10 points, and the total score for the last 2 questions is 18 points. Therefore, Xiao Ming's total score is ( ) points.",
        "answer": "35",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 129758,
        "difficulty": null
      },
      {
        "id": "4ea45a0622a67809b4828a92e629703e6adda17fa926e818a890d5ebb6be26b0",
        "domain": "numina",
        "question": "11. Toll Road (from 8th grade. 3 points). The cost of traveling along a section of a toll road depends on the class of the vehicle: passenger cars belong to the first class, for which the travel cost is 200 rubles, and light trucks and minivans belong to the second class, for which the cost is 300 rubles.\n\nAt the entrance to the toll booth, an automatic system is installed that measures the height of the vehicle. If the height of the vehicle is less than a certain threshold value \\( h \\), the system automatically classifies it as class 1; if it is higher than \\( h \\), it is classified as class 2. Errors are possible. For example, a low minivan may be classified as class 1 by the system, and its driver will be pleased. A low SUV may be incorrectly classified as class 2, and its driver will not be happy. The driver can file a claim, and the operating company will have to refund 100 rubles.\n\nThe company's management tasked the engineers with the goal of configuring the system so that the number of errors is minimized.\n\nFor several weeks, the engineers collected data on the height of vehicles passing through the toll booth. Separately for class 1 vehicles and separately for class 2 vehicles (Fig. 5). On the x-axis, the height of the vehicle (in cm) is plotted, and on the y-axis, the average number of vehicles of such height passing through the toll booth per day. For clarity, the points are connected by a smooth line.\n![](https://cdn.mathpix.com/cropped/2024_05_06_a6149daf2f3a04ecb66bg-05.jpg?height=486&width=1628&top_left_y=274&top_left_x=266)\n\nFig. 5. Graphs - the number of class 1 and class 2 vehicles per day\n\nHaving gathered all this information and drawn the graphs, the engineers began to think about what to do next: how to determine the threshold value \\( h \\) so that the probability of error is the smallest possible? Solve this problem for them.",
        "answer": "190",
        "topic": "Other",
        "source": "olympiads",
        "source_index": 11058,
        "difficulty": null
      },
      {
        "id": "27073dd024c51ea9f6fb8340da5cf0ef2495ebed63106256fa9c83b9a7a36511",
        "domain": "numina",
        "question": "Problem 9.8. On the side $CD$ of trapezoid $ABCD (AD \\| BC)$, a point $M$ is marked. A perpendicular $AH$ is dropped from vertex $A$ to segment $BM$. It turns out that $AD = HD$. Find the length of segment $AD$, given that $BC = 16$, $CM = 8$, and $MD = 9$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d55ea3e8288d58a3f771g-35.jpg?height=444&width=589&top_left_y=743&top_left_x=432)",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15183,
        "difficulty": null
      },
      {
        "id": "15bfd3273b3345fab18fd6b5c0efa46c229d85bb70a74efec0244c3e4b75ce04",
        "domain": "numina",
        "question": "I am Ivan and I am three times younger than my father. I have brothers Vincent and Jakub, who are 11 and 9 years old. My age is equal to five times the third of the age of the younger of the brothers. A peculiarity of our family is that we were all born on April 12th, so today we are celebrating our birthdays.\n\nHow many years will it take for the age of our father to be equal to the sum of the ages of the three of us? (L. Hozová)",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 76894,
        "difficulty": null
      },
      {
        "id": "ac93831d491596ab4423be4fc021bbfc900b991e89e59feaec52a0c0e06308ea",
        "domain": "numina",
        "question": "4. The integer part of the result of the expression $0.81 \\times 0.6+0.83 \\times 0.6+\\cdots+0.97 \\times 0.6+0.99 \\times 0.6$ is\n$\\qquad$ .",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 105734,
        "difficulty": null
      },
      {
        "id": "c9df1f5ad98f3aa127a8d75ba91a84aa1c25566ff455c63c55ffff39e65f9ff9",
        "domain": "numina",
        "question": "By how much should ${ }^{4} \\log 8$ be multiplied to obtain ${ }^{32} \\log 8$?",
        "answer": "\\frac{2}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 173176,
        "difficulty": null
      },
      {
        "id": "75dad4fdbcd4f1a2171d763fb36ce8cb43feba3e558a11e00f89d5ea25689752",
        "domain": "numina",
        "question": "6. In the Cartesian coordinate system $x O y$, the point set $K=\\{(x, y) \\mid x, y=-1,0,1\\}$. If three points are randomly selected from $K$, then the probability that the distance between each pair of these three points is no more than 2 is $\\qquad$ .",
        "answer": "\\frac{5}{14}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 141366,
        "difficulty": null
      },
      {
        "id": "8c78fedfb1b140475386e18c497714cda9e2529beab31d047d90354d005f9be8",
        "domain": "numina",
        "question": "In 8th grade class \"G\", there are enough underachievers, but Vovochka studies the worst of all. The pedagogical council decided that either Vovochka must correct his twos by the end of the quarter, or he will be expelled. If Vovochka corrects his twos, then 24% of the class will be underachievers, and if he is expelled, the underachievers will become 25%. What percentage of underachievers are there in 8 \"G\" now? #",
        "answer": "28",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69455,
        "difficulty": null
      },
      {
        "id": "10747f1762fe091f6e4097e8c5c3c6df32a6ad919cae8dc9c87093664f7c01d8",
        "domain": "numina",
        "question": "We consider those natural numbers to be somewhat unlucky whose sum of digits in decimal notation is divisible by 13. What is the maximum difference between consecutive somewhat unlucky numbers?",
        "answer": "79",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 168939,
        "difficulty": null
      },
      {
        "id": "f8f2ea3f8db927fc0533ae32f046ca92256d6c37406bb5703b4880a0878e8741",
        "domain": "numina",
        "question": "10. $f(x)=\\frac{2}{4^{x}+2}+2$, then $f\\left(\\frac{1}{m}\\right)+f\\left(\\frac{2}{m}\\right)+\\cdots+f\\left(\\frac{m-1}{m}\\right)=$ $\\ldots(m \\geqslant 2, m \\in \\mathbf{N})$.",
        "answer": "\\frac{5}{2}(-1)",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 145641,
        "difficulty": null
      },
      {
        "id": "e480805b0fe4c8f7e83575ebd6b16c746fb8cf531dad1f9a048dbe9b0b1b45a3",
        "domain": "numina",
        "question": "Problem 9. In a convex quadrilateral $A B C D$, side $A B$ is equal to diagonal $B D, \\angle A=65^{\\circ}$, $\\angle B=80^{\\circ}, \\angle C=75^{\\circ}$. What is $\\angle C A D$ (in degrees $) ?$",
        "answer": "15",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 4982,
        "difficulty": null
      },
      {
        "id": "bedaee8b70ca78e52d42d5c54a83918c5efba6614455234b6e5d3e89c148a141",
        "domain": "numina",
        "question": "3. Arrange the numbers in the set $\\left\\{2^{x}+2^{y} \\mid x 、 y \\in \\mathbf{N}, x<y\\right\\}$ in ascending order. Then the 60th number is $\\qquad$ (answer with a number).\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "2064",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 105615,
        "difficulty": null
      },
      {
        "id": "ee20b0257df13f296f208bab74e279af7885963ca28d88e0d503284a051ecae2",
        "domain": "numina",
        "question": "1. Find all prime numbers $p$ such that\n$$\n1+2^{p}+3^{p}+\\cdots+p^{p}\n$$\n\nis a prime number.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 99906,
        "difficulty": null
      },
      {
        "id": "c9ffc6bae7a72cd1e44c689887e14663a431a824bb0a09ca88fe6e7d5e335619",
        "domain": "numina",
        "question": "Example 9 Let real numbers $s, t$ satisfy the equations $19 s^{2}+99 s+1=0, t^{2}+99 t+19=0$, and $s t \\neq 1$.\nThen $\\frac{s t+4 s+1}{t}=$ $\\qquad$ .",
        "answer": "-5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129937,
        "difficulty": null
      },
      {
        "id": "e912ead4aabe54505b614d155c7ec3064e25178484a90ad3cbedb0ddc2078185",
        "domain": "numina",
        "question": "7. In tetrahedron $A B C D$, $\\angle A D B=\\angle B D C$ $=\\angle C D A=60^{\\circ}, A D=B D=3, C D=2$. Then the volume of the circumscribed sphere of tetrahedron $A B C D$ is $\\qquad$ .",
        "answer": "4\\sqrt{3}\\pi",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 145826,
        "difficulty": null
      },
      {
        "id": "6b8dfacafa93dacf9cbfe13ebe13fb979d9fdd75a0ae80b411ff1eb12c502e98",
        "domain": "numina",
        "question": "5. In an $m \\times 10$ grid, each cell is filled with either 0 or 1, such that each column contains exactly three 1s. The sum of the numbers in the $i(i=1,2, \\cdots, m)$-th row is denoted as $x_{i}$, and for any two columns, there always exists a row where the cells at the intersection with these two columns are both filled with 1. Let $x=\\max _{1 \\leqslant i \\leqslant m}\\left\\{x_{i}\\right\\}$. Then $x_{\\text {min }}=$",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 131134,
        "difficulty": null
      },
      {
        "id": "cd216505d2661ac0ac518d83d62a0d20167b8166135e2dae77b0b2a45cb24292",
        "domain": "numina",
        "question": "7. (10 points) If $2 \\times 3^{8}$ can be expressed as the sum of $k$ consecutive positive integers, then the maximum value of $k$ is\n\nExpress the above text in English, keeping the original text's line breaks and format, and output the translation result directly.",
        "answer": "108",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 96356,
        "difficulty": null
      },
      {
        "id": "bbe268da90ae922ed474b2ebf0f40aa1e8f7f575b97a7c5d20abc947eaa9a7de",
        "domain": "numina",
        "question": "13. (15 points) A speedboat departs from dock $A$, travels downstream along the river, passes through dock $B$, and continues downstream to dock $C$. Upon reaching dock $C$, it immediately turns around and heads back to dock $B$, taking a total of 10 hours. If the distance between $A$ and $B$ is 20 kilometers, the speed of the speedboat in still water is 40 kilometers/hour, and the speed of the river current is 10 kilometers/hour, find the distance between $B$ and $C$.",
        "answer": "180",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 135691,
        "difficulty": null
      },
      {
        "id": "98858e1715981d51fc091a366f75102dfd6bda5a3df050900f40c322f9f53507",
        "domain": "numina",
        "question": "Find the minimum of the function\n\n$$\nf(x, y)=\\sqrt{(x+1)^{2}+(2 y+1)^{2}}+\\sqrt{(2 x+1)^{2}+(3 y+1)^{2}}+\\sqrt{(3 x-4)^{2}+(5 y-6)^{2}} \\text {, }\n$$\n\ndefined for all real $x, y>0$.",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 90239,
        "difficulty": null
      },
      {
        "id": "5007bc1a22919c5c9c8859b907d76ca8fa959cd82cfc0614d7b0e087155f2ad5",
        "domain": "numina",
        "question": "6. Let real numbers $a, b, c$ satisfy $a^{2}+b^{2}+c^{2}=1$, and let the maximum and minimum values of $ab+bc+ca$ be $M, m$ respectively. Then $M-m=$ $\\qquad$ .",
        "answer": "\\frac{3}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 107115,
        "difficulty": null
      },
      {
        "id": "e9d3cff7d96fce54d809edd02307d75b5c7220fad9d69219f4bb71acecf1edc2",
        "domain": "numina",
        "question": "Esquecinaldo has a terrible memory for remembering numbers, but excellent for remembering sequences of operations. Therefore, to remember his five-digit bank code, he can remember that the code does not have repeated digits, none of the digits is zero, the first two digits form a power of 5, the last two digits form a power of 2, the middle digit is a multiple of 3, and the sum of all the digits is an odd number. Now he no longer needs to memorize the number, because he knows that his code is the largest number that satisfies these conditions. What is this code?",
        "answer": "25916",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 75003,
        "difficulty": null
      },
      {
        "id": "c1a400a52c8e43c86dd0b53718f1e5a64ba07d1e5653c50b715750125bfd1127",
        "domain": "numina",
        "question": "21. Ten people exchanged photographs. How many photographs were needed?",
        "answer": "90",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 44842,
        "difficulty": null
      },
      {
        "id": "109304d634c056e7551bccf7ddd2ef8a38afcc41e91a5b52fd1eafc343924bd7",
        "domain": "numina",
        "question": "4. On a circle, 100 points are marked, painted either red or blue. Some of the points are connected by segments, with each segment having one blue end and one red end. It is known that no two red points belong to the same number of segments. What is the maximum possible number of red points?",
        "answer": "50",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5835,
        "difficulty": null
      },
      {
        "id": "ce2c96709474e53bc6e254b6e9bab611eac96a114136403b91d9d784033bc5a7",
        "domain": "numina",
        "question": "## Task 4.\n\nDetermine all natural numbers $n \\geqslant 3$ for which the product of the first $n$ natural numbers divides the product of all sums of distinct pairs of prime numbers not greater than $n$, i.e., for which\n\n$$\nn! \\mid \\prod_{\\substack{p<q \\leqslant n \\\\ p, q \\text{ prime }}}(p+q)\n$$",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 89688,
        "difficulty": null
      },
      {
        "id": "e382466425ef2bcca78d9b0a452a21533ecfdd03aa4f929d45eb7f5623ec414b",
        "domain": "numina",
        "question": "19. Erin lists all three-digit primes that are 21 less than a square. What is the mean of the numbers in Erin's list?",
        "answer": "421",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 156429,
        "difficulty": null
      },
      {
        "id": "ef9f742832ae2b35ff2984210d7c97115564bf2b8cd8929b8a71f145f2bfed50",
        "domain": "numina",
        "question": "Every day, the dog Patrick gnaws one slipper from the available supply in the house. With a probability of 0.5, Patrick wants to gnaw a left slipper and with a probability of 0.5 - a right slipper. If the desired slipper is not available, Patrick gets upset. How many pairs of identical slippers need to be bought so that with a probability of at least 0.8, Patrick does not get upset for a whole week (7 days)?",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55905,
        "difficulty": null
      },
      {
        "id": "6caad07f563e834a79e9973579bd8e5bb8a4a97e7fa5229d252cf228b218a902",
        "domain": "numina",
        "question": "5.1. (12 points) The decreasing sequence $a, b, c$ is a geometric progression, and the sequence $19 a, \\frac{124 b}{13}, \\frac{c}{13}$ is an arithmetic progression. Find the common ratio of the geometric progression.",
        "answer": "247",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6147,
        "difficulty": null
      },
      {
        "id": "ab15ca72d32acc7677a665a1b503f2bcb80a31db590196f55b70b7c5f9cf7822",
        "domain": "numina",
        "question": "3. Find the least positive number $x$ with the following property: if $a, b, c, d$ are arbitrary positive numbers whose product is 1 , then\n\n$$\na^{x}+b^{x}+c^{x}+d^{x} \\geqslant \\frac{1}{a}+\\frac{1}{b}+\\frac{1}{c}+\\frac{1}{d} .\n$$",
        "answer": "3",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 77204,
        "difficulty": null
      },
      {
        "id": "bb45b3ea159122c2399c2f3efdc816b1c0de090857d89f3bbd3b2fe43dabb363",
        "domain": "numina",
        "question": "18. The Absent-Minded Scientist on the Bus (4 p. 6-11). In the bus, there are $n$ seats, and all tickets have been sold to $n$ passengers. The first to enter the bus is the Absent-Minded Scientist, who, without looking at the ticket, takes the first available seat. Subsequently, passengers enter one by one. If an entering passenger sees that their seat is free, they take their seat. If the seat is occupied, the entering passenger takes the first available free seat. Find the probability that the last passenger to enter will take the seat according to their ticket?",
        "answer": "\\frac{1}{2}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10695,
        "difficulty": null
      },
      {
        "id": "9e843041760ec99d810c3aeda7cddf7d70722d8cc733b5d63aa765028dee850f",
        "domain": "numina",
        "question": "(12) $O, A, B$ are three non-collinear points on a plane, vector $\\overrightarrow{O A}=\\vec{a}, \\overrightarrow{O B}=\\vec{b}$, let $P$ be any point on the perpendicular bisector of line segment $A B$, vector $\\overrightarrow{O P}=\\vec{p}$. If $|\\vec{a}|=5,|\\vec{b}|=3$, then the value of $\\vec{p} \\cdot(\\vec{a}-\\vec{b})$ is $\\qquad$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 127859,
        "difficulty": null
      },
      {
        "id": "c61f888ba07ae690f200576af5c8e1aa96222274b57d3ebfb96877cddef2134d",
        "domain": "numina",
        "question": "13. Move 1 matchstick to make the equation in the figure correct. Write the equation after moving on the line:\n$\\qquad$ (Note: The \" $\\mathrm{Z}$ \" in the equation represents 2 )\n\n\n\nPlease write the equation after moving the matchstick on the line:",
        "answer": "11+1-2-7",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 142204,
        "difficulty": null
      },
      {
        "id": "ca581606ca97d6a60af78555f9dd7245b516a7fe8fe1fe8cf24385dbb00f1638",
        "domain": "numina",
        "question": "11. Define $a \\oplus b=(a+2)(b+2)-2$. The calculation result of $1 \\times 3 \\times 5 \\times 7 \\times 9 \\times 11 \\times 13-(1 \\bigcirc 30507 \\sim 9011)$ is $\\qquad$",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99472,
        "difficulty": null
      },
      {
        "id": "0968f7fb9f6457360f8531e14efb81d2a6d5d62636b5c6de29cad0281a61aa87",
        "domain": "numina",
        "question": "I4.1 若 $m$ 和 $n$ 為正整數及 $a=\\log _{2}\\left[\\left(\\frac{m^{4} n^{-4}}{m^{-1} n}\\right)^{-3} \\div\\left(\\frac{m^{-2} n^{2}}{m n^{-1}}\\right)^{5}\\right]$, 求 $a$ 的值。\nIf $m$ and $n$ are positive integers and $a=\\log _{2}\\left[\\left(\\frac{m^{4} n^{-4}}{m^{-1} n}\\right)^{-3} \\div\\left(\\frac{m^{-2} n^{2}}{m n^{-1}}\\right)^{5}\\right]$, determine the value of $a$.",
        "answer": "0",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153408,
        "difficulty": null
      },
      {
        "id": "3d4c9267d3fa4921ae1f9fed2ebf0bbe8764d723b52932e04e3c565a5577f19d",
        "domain": "numina",
        "question": "[ Height of a pyramid (tetrahedron).]\n\nThe base of the pyramid is a right triangle with a hypotenuse of 6 and an acute angle of \\(15^{\\circ}\\). All lateral edges are inclined to the plane of the base at an angle of \\(45^{\\circ}\\). Find the volume of the pyramid.",
        "answer": "4.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 62014,
        "difficulty": null
      },
      {
        "id": "742d30d421af9a807f6c5961e2064b8b1697fa245545b4c064b35186aabf59d9",
        "domain": "numina",
        "question": "117. Find the minimum value of the expression\n\n$$\n(x-5)(x-1)(x-6)(x-2)+9\n$$",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 36041,
        "difficulty": null
      },
      {
        "id": "66c9bd88bbcb9e43fb6a60d03d435934d58a1f609e3bb73e4d9102a5950d5afd",
        "domain": "numina",
        "question": "3. The equation $a x^{2}-b x+c=0$ has two distinct roots $x_{1}, x_{2}$. What is the maximum number of numbers in the set $\\left\\{a, b, c, x_{1}, x_{2}\\right\\}$ that can be prime (and, accordingly, natural) numbers? If a prime number appears in the set twice, it should be counted twice. The number 1 is not considered a prime number.",
        "answer": "4",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 17014,
        "difficulty": null
      },
      {
        "id": "8cc05551913a921f7c51a68ff615eb80e31a590d431f7e9ec23e770f78bfa82e",
        "domain": "numina",
        "question": "[Text problems $]$\n\nLiza is 8 years older than Nastya. Two years ago, she was three times as old as Nastya. How old is Liza?\n\n#",
        "answer": "14",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 69013,
        "difficulty": null
      },
      {
        "id": "a9d8ae6b559d98ae99b31d300ff9531c4cc5a8f7073bb6a65a3853b43021babc",
        "domain": "numina",
        "question": "## Problem 4\n\nIn the cube $\\mathrm{ABCDA}^{\\prime} \\mathrm{B}^{\\prime} \\mathrm{C}^{\\prime} \\mathrm{D}^{\\prime}$, points $\\mathrm{M}, \\mathrm{N}$, and $\\mathrm{P}$ are the midpoints of the edges $\\mathrm{CC}$ ', $\\mathrm{A}^{\\prime} \\mathrm{D}^{\\prime}$, and $\\mathrm{C}^{\\prime} \\mathrm{D}^{\\prime}$, respectively. Find the sine of the angle formed by the lines BM and NP.\n\nProblems selected by Lung Ioan and Nicoara Florin\n\nNote: a) The actual working time is 3 hours.\n\nb) All problems are mandatory.\n\nc) Each problem is scored from 0 to 7.\n\n## NATIONAL MATHEMATICS OLYMPIAD\n\nLocal stage - 15.02.2014\n\n## Grade 8\n\n## Grading Rubric",
        "answer": "\\frac{\\sqrt{3}}{\\sqrt{5}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 179609,
        "difficulty": null
      },
      {
        "id": "ba62e1786a5d94816bc3f20dcb8ff450a7b2f0f5d886be27d4fb34924792a869",
        "domain": "numina",
        "question": "[The ratio of the areas of triangles with a common base or common height] [The area of a figure is equal to the sum of the areas of the figures into which it is divided\n\nPoints $M$ and $N$ are located on side $A C$ of triangle $A B C$, and points $K$ and $L$ are on side $A B$, such that $A M: M N: N C=1: 3: 1$ and $A K=K L=L B$. It is known that the area of triangle $A B C$ is 1. Find the area of quadrilateral $K L N M$.",
        "answer": "\\frac{7}{15}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 66278,
        "difficulty": null
      },
      {
        "id": "cc72f8137e0882d748128fa934ee22f7aa7ebb9f3646b7e71a06a0d51567e4e4",
        "domain": "numina",
        "question": "A round table has a diameter of $1.40 \\, \\text{m}$. For a party, the table is expanded by adding three planks, each $40 \\, \\text{cm}$ wide, as shown in the figure. If each person at the table should have a space of $60 \\, \\text{cm}$, how many guests can sit at the table?\n![](https://cdn.mathpix.com/cropped/2024_05_01_bd5219fde2a4a59a7005g-2.jpg?height=506&width=724&top_left_y=618&top_left_x=996)",
        "answer": "11",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 74217,
        "difficulty": null
      },
      {
        "id": "dbf5a7e6ba6a8f8d56a7b8374f89ae0bb372f82c2d535d2887729c4fff503692",
        "domain": "numina",
        "question": "1. If the sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=\\frac{1}{2}, a_{n+1}=\\frac{2 a_{n}}{3 a_{n}+2}, n \\in \\mathbf{N}_{+}$, then $a_{2017}=$ $\\qquad$ .",
        "answer": "\\frac{1}{3026}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 143226,
        "difficulty": null
      },
      {
        "id": "4fd6777f8f89a5b8422d0480a88f4029270f9d86d89c74e5d788d2f38a2815d8",
        "domain": "numina",
        "question": "4. In how many ways can seven different items (3 weighing 2 tons each, 4 weighing 1 ton each) be loaded into two trucks with capacities of 6 tons and 5 tons, if the arrangement of the items inside the trucks does not matter? (12 points)\n\n#",
        "answer": "46",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10292,
        "difficulty": null
      },
      {
        "id": "08790be316c0c9291276df4625f488b5413eb8a2af68ac356836bcc9b3d93527",
        "domain": "numina",
        "question": "$[$ Mathematical logic (miscellaneous).]\n\nAuthors: Folklore, Folklore\n\nIn a school of witchcraft, there are 13 students. Before the clairvoyance exam, the teacher seated them at a round table and asked them to guess who would receive the clairvoyant diploma. They modestly kept quiet about themselves and their two neighbors, but wrote about everyone else: \"None of these ten will get it!\" Of course, all those who passed the exam guessed correctly, while all the others were wrong. How many wizards received the diploma?\n\n#",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 55981,
        "difficulty": null
      },
      {
        "id": "380a251ac724a6879e79b0c7936b485d1bba9b775a0059734e1bbc4d4bed151c",
        "domain": "numina",
        "question": "14. There is a ten-digit number. Counting from left to right, if its first digit is a certain number, then this ten-digit number contains that many 0s; if its second digit is a certain number, then this ten-digit number contains that many 1s; if its third digit is a certain number, then this ten-digit number contains that many 2s; and so on; if its tenth digit is a certain number, then this ten-digit number contains that many 9s. This ten-digit number is $\\qquad$",
        "answer": "6210001000",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 98841,
        "difficulty": null
      },
      {
        "id": "1070a8872058df0d0e51d7ac02aba652d2d3700721991bda7ea264c25dedd9e1",
        "domain": "numina",
        "question": "## 5. Black and White Balls\n\nAn urn contains 5 white and 5 black balls. Then, 3 balls are randomly drawn one by one. This is done in two ways: 1 - the drawn ball is returned to the urn, 2 - the balls are not returned. In which of these two cases is there a higher probability of drawing one white and two black balls?",
        "answer": "\\frac{10}{24}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 41983,
        "difficulty": null
      },
      {
        "id": "fc9abf9d9bcd10ba1c42a364a4ff5c61d7326d279593fa2c9058754a84824e9f",
        "domain": "numina",
        "question": "1. Calculate in the most rational way\n\n$$\n12 \\times 171+29 \\times 9+171 \\times 13+29 \\times 16\n$$",
        "answer": "5000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42755,
        "difficulty": null
      },
      {
        "id": "2d6a0cb5c384498b04ae469659438e2fca100a46fdf4969ec1699aea1824446f",
        "domain": "numina",
        "question": "[ Regular pyramid ] [ Sections, unfoldings, and other frameworks. ]\n\nOn the extension of edge $S T$ beyond point $T$ of the regular quadrilateral pyramid $S P Q R T$ with vertex $S$, a point $B$ is taken such that the distance from it to the plane $S P Q$ is $\\frac{9 \\sqrt{7}}{2}$. Find the segment $B T$, if $Q R=12$, and $S R=10$.",
        "answer": "5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 63444,
        "difficulty": null
      },
      {
        "id": "01d802d0c690dc038b14a502aeb2285c413567c9798dcb2675b748d8bdb09c10",
        "domain": "numina",
        "question": "10. Determine the units digit of $(2!+2)(3!+3)(4!+2)(5!+3) \\cdots(2018!+2)(2019!+3)$.",
        "answer": "8",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 154677,
        "difficulty": null
      },
      {
        "id": "30e1b09381ed084bd1125e3240bf0e3c90bd582bf65446d6a41fa156fc9cd279",
        "domain": "numina",
        "question": "## Task B-1.5.\n\nInside an equilateral triangle $ABC$ with side length $12 \\, \\text{cm}$, a point $T$ is chosen such that $|TP|:|TQ|:|TR|=1:2:3$, where $P, Q$, and $R$ are the feet of the perpendiculars from point $T$ to the sides of the triangle. What is the area of triangle $\\triangle PQR$?",
        "answer": "\\frac{33\\sqrt{3}}{4}^2",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89517,
        "difficulty": null
      },
      {
        "id": "f048b95adc6f87c5ee08a22ed29d88bd383b4751f915f9d0837963c137ac81eb",
        "domain": "numina",
        "question": "A square has side length $s$ and a diagonal of length $s+1$. Write the area of the square in the form $a+b \\sqrt{c}$ where $a, b$ and $c$ are positive integers.",
        "answer": "3+2\\sqrt{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194147,
        "difficulty": null
      },
      {
        "id": "340cc5670b18c2d7b30167b1c8a269fdb99e96ebda834615ecf55f30257abda9",
        "domain": "numina",
        "question": "13. (12 points) Cars A and B start from point A to point B at the same time. At the start, car A's speed is 2.5 kilometers per hour faster than car B's. After 10 minutes, car A slows down; 5 minutes later, car B also slows down, at which point car B is 0.5 kilometers per hour slower than car A. Another 25 minutes later, both cars arrive at point B at the same time. By how many kilometers per hour did car A's speed decrease at that time?",
        "answer": "10",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 99840,
        "difficulty": null
      },
      {
        "id": "4f6491116a717de98b08d09b7971c633d81a6a29848197f0f689c2718aa3b6f3",
        "domain": "numina",
        "question": "1. For a set of real numbers $A$, let $A-A$ be the set of distinct pairwise differences of $A$; that is,\n$$\nA-A:=\\{a-b: a, b \\in A\\}\n$$\n\nIf $|A-A|=25$, find the sum of all possible values of $|A|$.",
        "answer": "76",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 154593,
        "difficulty": null
      },
      {
        "id": "24d898c4bcf0880e660ede0af7232983ab4db01471d96415e95188e97d222504",
        "domain": "numina",
        "question": "7. Given the polynomial $P(x)=x^{5}+a_{4} x^{4}+a_{3} x^{3}+a_{2} x^{2}+a_{1} x+a_{0}$, it is known that $P(2014)=1, P(2015)=2, P(2016)=3, P(2017)=4$, $P(2018)=5$. Find $P(2013)$.",
        "answer": "-120",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6042,
        "difficulty": null
      },
      {
        "id": "9ab5d59a941724f7e0a8efd1de22e2ca8e510b62edd699638ea4a295f41a3c59",
        "domain": "numina",
        "question": "We denote by $\\mathbb{Z}[x, y, z]$ the set of polynomials with integer coefficients in the three variables $x, y$, and $z$. We then say that a polynomial $P \\in \\mathbb{Z}[x, y, z]$ is Olympic if $\\mathbb{Z}[x, y, z]$ contains polynomials $A, B$, and $C$ such that\n\n$$\nP(x, y, z)=(x+y+z) A(x, y, z)+(x y+y z+z x) B(x, y, z)+x y z C(x, y, z).\n$$\n\nFind the largest integer $n$ for which there exist natural numbers $i, j$, and $k$ with sum $i+j+k=n$ and such that the polynomial $x^{i} y^{j} z^{k}$ is not Olympic.\nNote: A polynomial with integer coefficients in the variables $x, y$, and $z$ is a function that can be written as a sum of terms of the form $\\lambda x^{i} y^{j} z^{k}$, where $\\lambda$ is an integer and $i, j$, and $k$ are natural numbers. For example, $x-y+1$ and $x y+y z+z x$ are such polynomials, but $\\pi x y z, \\exp (x), x /\\left(y^{2}+1\\right)$, and $\\sqrt{x y+z}$ are not.",
        "answer": "3",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 195728,
        "difficulty": null
      },
      {
        "id": "63c74c2a4fe89a080ca425763c6f9593a37c027e900fae360bf9499efe7a8e59",
        "domain": "numina",
        "question": "The points $P(2,0), Q(11,-3)$ and $R(x, 3)$ are the vertices of a triangle with $\\angle P Q R=90^{\\circ}$. What is the value of $x$ ?",
        "answer": "13",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 194376,
        "difficulty": null
      },
      {
        "id": "a97023da908d7f5671e5bfc8350b1bed0dc629d8c95b9d56f142fa496577af71",
        "domain": "numina",
        "question": "(10 points) Calculate: $2016 \\times 2016 - 2015 \\times 2016=$",
        "answer": "2016",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 98074,
        "difficulty": null
      },
      {
        "id": "c0657876b06f39ef3308e6fb80d9de3293ae0250b5301510cb88a01ac2cab8be",
        "domain": "numina",
        "question": "1. Let $A B C$ be a triangle with $\\angle A=60^{\\circ}$. Line $\\ell$ intersects segments $A B$ and $A C$ and splits triangle $A B C$ into an equilateral triangle and a quadrilateral. Let $X$ and $Y$ be on $\\ell$ such that lines $B X$ and $C Y$ are perpendicular to $\\ell$. Given that $A B=20$ and $A C=22$, compute $X Y$.",
        "answer": "21",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151928,
        "difficulty": null
      },
      {
        "id": "c1dd453bb802329d0bc220d1d872461cfd27dc9dbf486e89759810bf024bf027",
        "domain": "numina",
        "question": "3. Compute the greatest common divisor of $4^{8}-1$ and $8^{12}-1$.",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151540,
        "difficulty": null
      },
      {
        "id": "e193cc21a9910cc5f32a28da0236dd94cb7b1c6b3206f803b659c0e1134026d3",
        "domain": "numina",
        "question": "1537. In an urn, there are 5 white and 4 black balls. Two balls are drawn in succession. Find the probability that both balls are white.",
        "answer": "\\frac{5}{18}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 30643,
        "difficulty": null
      },
      {
        "id": "f0ae55b97a8a60c9f35f8a438865bae8556d855150ffc13c1855972593f88e5a",
        "domain": "numina",
        "question": "1. [2] What is the smallest non-square positive integer that is the product of four prime numbers (not necessarily distinct)?",
        "answer": "24",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 151485,
        "difficulty": null
      },
      {
        "id": "475824ad724e2796c2dbd64177919970efe7c70865ed9490a400c2c91a5ec1ce",
        "domain": "numina",
        "question": "9. Let $F$ be the left focus of the ellipse $E: \\frac{x^{2}}{3}+y^{2}=1$. A line $l$ passing through point $F$ with a positive slope intersects $E$ at points $A$ and $B$. Lines $A M$ and $B N$ are drawn through points $A$ and $B$ respectively, such that $A M \\perp l, B N \\perp l$, and lines $A M, B N$ intersect the $x$-axis at points $M$ and $N$. Find the minimum value of $|M N|$.",
        "answer": "\\sqrt{6}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133148,
        "difficulty": null
      },
      {
        "id": "a8896584472b8e10a500a731e70caad825297e278eb5dbf0f0160a866a2f4f8e",
        "domain": "numina",
        "question": "9. Team A and Team B each send out 7 players to participate in a Go chess tournament according to a pre-arranged order. Both sides start with the No. 1 player competing, the loser is eliminated, and the winner then competes with the No. 2 player of the losing side, ..., until all players of one side are eliminated, and the other side wins, forming a match process. Then the total number of all possible match processes is $\\qquad$ .",
        "answer": "3432",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 145284,
        "difficulty": null
      },
      {
        "id": "28ce2920fd974d36e05ef56be43c428a7ac8ee3dbbc4365e1635e425f300af86",
        "domain": "numina",
        "question": "## Task B-4.3.\n\nHow many isosceles trapezoids, with bases of different lengths, have integer side lengths and a perimeter of 2010?",
        "answer": "251502",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 88049,
        "difficulty": null
      },
      {
        "id": "aa857d573e1b8a318780ffd02b70167fa80d12b0413be6fdc8bb735eda763310",
        "domain": "numina",
        "question": "5. There were 5 times more strawberry bushes on the first bed than on the second. When 22 bushes were transplanted from the first bed to the second, the number of strawberry bushes on each bed became the same. How many bushes were there on each bed?",
        "answer": "11",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42681,
        "difficulty": null
      },
      {
        "id": "1fe941d9765dc0d808104352043b4801dfb2c752789ee78b8ea503924a1081e5",
        "domain": "numina",
        "question": "Example 6 Given the function $f(x)=x^{2}-a x+\\frac{a}{2}(a>0), x \\in[0,1]$, find the minimum value $g(a)$ of $f(x)$, and then find the maximum value of $g(a)$.",
        "answer": "\\frac{1}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 132597,
        "difficulty": null
      },
      {
        "id": "0b4ec0b4bf4ec9b15aee9ae118993870d72639a8318259252a271ec94db8b722",
        "domain": "numina",
        "question": "Task 5. (20 points) In the center of a circular field stands a geologists' cabin. From it, 6 straight roads extend, dividing the field into 6 equal sectors. Two geologists set out on a journey from their cabin at a speed of 4 km/h along a road each arbitrarily chooses. Determine the probability that the distance between them after one hour will be at least 6 km.\n\n#",
        "answer": "0.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 7384,
        "difficulty": null
      },
      {
        "id": "7819da13e3ff0518e762d0ea186b54d4200371cffc74661343e3eb17076d786c",
        "domain": "numina",
        "question": "1. Each of the 5 brothers owns a plot of land. One day, they pooled their money, bought a neighbor's garden, and divided the new land equally among themselves. As a result, Andrey's plot increased by $10 \\%$, Boris's plot - by $\\frac{1}{15}$, Vladimir's plot - by $5 \\%$, Grigory's plot - by $4 \\%$, and Dmitry's plot - by $\\frac{1}{30}$. By what percentage did the total area of their land increase as a result?",
        "answer": "5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 15557,
        "difficulty": null
      },
      {
        "id": "00bd9261b70ce5568d2b9c166b82ed3bd55152c151ea966b2b2227a4032a1a1c",
        "domain": "numina",
        "question": "Segment $A B$ is the diameter of a circle. A second circle with center at point $B$ has a radius of 2 and intersects the first circle at points $C$ and $D$. Chord $C E$ of the second circle is part of a tangent to the first circle and is equal to 3. Find the radius of the first circle.",
        "answer": "\\frac{4}{\\sqrt{7}}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 61913,
        "difficulty": null
      },
      {
        "id": "b8affbfa7b5c2d14cd34ce053082123959f1cbb137450e4aaf8cd7a6ef85a5c5",
        "domain": "numina",
        "question": "30. If $\\sin \\theta-\\cos \\theta=\\frac{\\sqrt{6}-\\sqrt{2}}{2}$, find the value of $24\\left(\\sin ^{3} \\theta-\\cos ^{3} \\theta\\right)^{2}$,",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 160176,
        "difficulty": null
      },
      {
        "id": "b66607a0e8f6b4aa70fa6527395d9b9ce0b5ca9231bbf43477565b5430dd56a5",
        "domain": "numina",
        "question": "According to the conditions of the chess match, the winner is the one who outperforms the opponent by two wins. Draws do not count. The probabilities of winning for the opponents are equal. The number of decisive games in such a match is a random variable. Find its mathematical expectation.",
        "answer": "4",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 55865,
        "difficulty": null
      },
      {
        "id": "558d2c702cf5cd60834ad2ef0f75f23cee1be355850bf8573d278c4cf48551c0",
        "domain": "numina",
        "question": "3. Petrov and Vasechkin were repairing a fence. Each had to nail a certain number of boards (the same amount). Petrov nailed two nails into some boards, and three nails into the rest. Vasechkin nailed three nails into some boards, and five nails into the rest. Find out how many boards each of them nailed, if it is known that Petrov nailed 87 nails, and Vasechkin nailed 94 nails.",
        "answer": "30",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5034,
        "difficulty": null
      },
      {
        "id": "7b9378e452be188f70c4efaf5343d8a1492357c7daed871adbdfd59fc70c0dc5",
        "domain": "numina",
        "question": "8. Xiao Yu's family consists of Dad, Mom, Brother, Sister, and Xiao Yu, a total of five people. From the 1st to the 5th of this month, there are two household chores every day: one person cooks, and another person washes the dishes, and everyone has to cook once and wash the dishes once in these 5 days. One day they had the following conversation:\nBrother: I just washed the dishes yesterday, and I have to cook tomorrow, so tiring!\nSister: I am even more tired, I have chores both tomorrow and the day after!\nXiao Yu: Dad and I always do chores on the same day.\nDad: Yes, fortunately, after cooking, I can rest for two days before it's my turn to wash the dishes again.\nSo the five-digit number formed by the dates when Dad, Mom, Brother, Sister, and Xiao Yu wash the dishes in sequence is $\\qquad$",
        "answer": "54132",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 129465,
        "difficulty": null
      },
      {
        "id": "053bdae98a2025c814eb91cd15513ec6a3aeebfaa002191951340dae208a8048",
        "domain": "numina",
        "question": "1. Find all positive real numbers $x$ that satisfy the equation $x^{x \\sqrt[3]{x}}=(x \\sqrt[3]{x})^{x}$.",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 162612,
        "difficulty": null
      },
      {
        "id": "39debb9256e11770e45a1cf54902c8185fd223cf3a0fb8d97f61125a971b953d",
        "domain": "numina",
        "question": "12.11 $f(x)=\\frac{(x-2)^{2}}{x^{2}}$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 49982,
        "difficulty": null
      },
      {
        "id": "3dd7080e00965142c8fed6ee082b31125f926226873785720d706489ac616d1f",
        "domain": "numina",
        "question": "7.2. Find the sum of all three-digit natural numbers that do not contain the digits 0 or 9 in their representation.",
        "answer": "255744",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 22424,
        "difficulty": null
      },
      {
        "id": "56d9ee6327ad545479a29602fad4fff778bede18ab13cfbdde1e52602ab76d5f",
        "domain": "numina",
        "question": "For example, among 211 theater troupes, some troupes perform each day while others watch (those who perform cannot watch). If each troupe has seen performances by 10 other troupes, how many days are needed for the performances at least?\n\n\n\nTranslating the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
        "answer": "6",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 95565,
        "difficulty": null
      },
      {
        "id": "ac96c483ad1a7b9a58fd2475ea57a6f5549804b8fbba472fc4f2730e195fc13f",
        "domain": "numina",
        "question": "5. Point $A_{1}$ lies on side $B C$ of triangle $A B C$, point $B_{1}$ lies on side $A C$. Let $K$ be the intersection point of segments $A A_{1}$ and $B B_{1}$. Find the area of quadrilateral $A_{1} C B_{1} K$, given that the area of triangle $A_{1} B K$ is 5, the area of triangle $A B_{1} K$ is 8, and the area of triangle $A B K$ is 10.",
        "answer": "22",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 43219,
        "difficulty": null
      },
      {
        "id": "b8d60418b3019f6c3ce4ed56bbe50d0058816384e60cfc0f9d9692b216136a89",
        "domain": "numina",
        "question": "A palindrome is a positive integer that reads the same forward and backward, like 2552 or 1991. Find a positive integer greater than 1 that divides all four-digit palindromes.",
        "answer": "11",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 90900,
        "difficulty": null
      },
      {
        "id": "b1e5724501a3d60b3a305b99dac599933d3157f0c3fa44b1251243ade88bf27f",
        "domain": "numina",
        "question": "18. Lonely Cars. (From 9th grade, 4 points.) On a very long narrow highway, where overtaking is impossible, $n$ cars are driving in a random order, each with its own preferred speed. If a fast car catches up to a slower one, the fast car has to slow down and drive at the same speed as the slower one. Thus, the cars form groups. Find the expected number of \"lonely\" cars, that is, groups consisting of a single car.",
        "answer": "1",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 10841,
        "difficulty": null
      },
      {
        "id": "46da163f66166580f2d407bd98f68077c605afc8542960d02f73e4b1c1491c18",
        "domain": "numina",
        "question": "4. In the computer center, there are 200 computers, some of which (in pairs) are connected by cables, a total of 345 cables are used. We will call a \"cluster\" a set of computers such that a signal from any computer in this set can reach all other computers via cables (possibly through intermediate computers). Initially, all computers formed one cluster. But one night, a malicious hacker cut several cables, resulting in 8 clusters. Find the maximum possible number of cables that were cut.",
        "answer": "153",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 5340,
        "difficulty": null
      },
      {
        "id": "84a1883bcb231100c111583f6f2fae41c84de8b5b7112c68a318fb5d5066bc5c",
        "domain": "numina",
        "question": "146. Find $\\lim _{x \\rightarrow 0} \\frac{1-\\cos x}{x^{2}}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 31955,
        "difficulty": null
      },
      {
        "id": "193a46a0a58e444f23d004ebbb3a422b1dfe6419c0b9493df273fc543656ca21",
        "domain": "numina",
        "question": "20. Let $a, b, c, d$ be positive integers such that the least common multiple (L.C.M.) of any three of them is $3^{3} \\times 7^{5}$. How many different sets of $(a, b, c, d)$ are possible if the order of the numbers is taken into consideration?\n(2 marks)\n已知 $a 、 b 、 c 、 d$ 爲正整數, 當中任意三個數的最小公倍數 (L.C.M. ) 圴爲 $3^{3} \\times 7^{5}$ 。若需考虑各數的先後次序, 問 $(a, b, c, d)$ 有多少組不同的可能値?",
        "answer": "11457",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153592,
        "difficulty": null
      },
      {
        "id": "66d519a8b400586a8056ff9d25be5b4b00bcdd44e0377eb8d99f5729c39ea391",
        "domain": "numina",
        "question": "12. (NET) ${ }^{\\mathrm{IMO} 3}$ Determine the maximum value of $m^{2}+n^{2}$ where $m$ and $n$ are integers satisfying  \n$$ m, n \\in\\{1,2, \\ldots, 100\\} \\quad \\text { and } \\quad\\left(n^{2}-m n-m^{2}\\right)^{2}=1 $$",
        "answer": "987^2 + 1597^2",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 196842,
        "difficulty": null
      },
      {
        "id": "fdd27983d917e182a2ae7f2cf5359836f05e2ba2cb47b0040bb079031b9e9ad6",
        "domain": "numina",
        "question": "414. The line $L$ passes through the point $(2 ; 1 ;-1)$ and the point of intersection of the line\n\n$$\n\\frac{x-3}{2}=\\frac{y}{-1}=\\frac{z+1}{1}\n$$\n\nwith the plane\n\n$$\nx-y+z-1=0\n$$\n\nFind the angle formed by the line $L$ with the plane\n\n$$\nx+2 y-z+3=0\n$$",
        "answer": "197^{\\}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 30496,
        "difficulty": null
      },
      {
        "id": "dc1d3fdf228dbec786e168c1a77f862c81f6d257321f0dcfd063c082d1597ee1",
        "domain": "numina",
        "question": "9.5. Three hundred non-zero integers are written in a circle such that each number is greater than the product of the next three numbers in the clockwise direction. What is the maximum number of positive numbers that can be among these 300 written numbers?",
        "answer": "200",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 18550,
        "difficulty": null
      },
      {
        "id": "479cd281b1e2332653e91e3522fe5393c18f21cdf2f8c0f0c24c2725e4404bfd",
        "domain": "numina",
        "question": "(7) The function $f(x)=\\frac{\\sin \\left(x+45^{\\circ}\\right)}{\\sin \\left(x+60^{\\circ}\\right)}, x \\in\\left[0^{\\circ}, 90^{\\circ}\\right]$, then the product of the maximum and minimum values of $f(x)$ is . $\\qquad$",
        "answer": "\\frac{2\\sqrt{3}}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 141470,
        "difficulty": null
      },
      {
        "id": "2e8780eb7ade11218dd79d1b0971aaf4ae7303ac940b3934ea8e7a1994257781",
        "domain": "numina",
        "question": "11. Color the 17 squares in the \"T\" shape on the right using black and white. If each square can only be one color, and no two black squares can be adjacent (share an edge), then there are $\\qquad$ coloring schemes.\n\nPaint the 17 squares in the \"T\" shape on the right using black and white. If each square can only be one color, and no two black squares can be adjacent (share an edge), then there are $\\qquad$ coloring schemes.",
        "answer": "4370",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 101348,
        "difficulty": null
      },
      {
        "id": "e106d84db6955c39cb6f4a52539ee22004f3d5b9a87103c97e56a4b8c5b7fd3a",
        "domain": "numina",
        "question": "# 11.2 Solve the equation\n\n$$\n\\frac{1}{\\sqrt{x+20}+\\sqrt{x+21}}+\\frac{1}{\\sqrt{x+21}+\\sqrt{x+22}}=1\n$$",
        "answer": "-\\frac{79}{4}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 12821,
        "difficulty": null
      },
      {
        "id": "57b219ff8b93ba19d68854de928ae02ffbd1fbb1b85505ce7f42c8e3f24f7f24",
        "domain": "numina",
        "question": "10. (10 points) Convert the 2015 fractions $\\frac{1}{2}, \\frac{1}{3}, \\frac{1}{4} \\cdots \\frac{1}{2014}, \\frac{1}{2015}, \\frac{1}{2016}$ into decimals. How many of them are finite decimals?",
        "answer": "33",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103504,
        "difficulty": null
      },
      {
        "id": "5e3658742fc8ea93b98371853355f2b44841d71132d19c1f0f2abedf541a6bdd",
        "domain": "numina",
        "question": "3. (GDR 1) Knowing that the system \n\\[ \\begin{aligned} x+y+z & =3, \\\\ x^{3}+y^{3}+z^{3} & =15, \\\\ x^{4}+y^{4}+z^{4} & =35, \\end{aligned} \\]\nhas a real solution \\(x, y, z\\) for which \\(x^{2}+y^{2}+z^{2}<10\\), find the value of \\(x^{5}+y^{5}+z^{5}\\) for that solution.",
        "answer": "83",
        "topic": "Algebra",
        "source": "olympiads_ref",
        "source_index": 196692,
        "difficulty": null
      },
      {
        "id": "683f75e0aa7012358cea02c3539d7e9318d5c10f3a143e05a9fb92e20df05e23",
        "domain": "numina",
        "question": "2. Square $A B C D$ and square $A B E F$ are in planes that form a $120^{\\circ}$ angle, $M, N$ are points on the diagonals $A C$, $B F$ respectively, and $A M=F N$. If $A B=1$, then the maximum value of $M N$ is $\\qquad$ .",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 108815,
        "difficulty": null
      },
      {
        "id": "24b8b68b2fe859fd5a16ee1509a1d666cf6448ed6682121c549c6e54c30b7970",
        "domain": "numina",
        "question": "23. If $x$ is positive, find the minimum value of $\\frac{\\sqrt{x^{4}+x^{2}+2 x+1}+\\sqrt{x^{4}-2 x^{3}+5 x^{2}-4 x+1}}{x}$. If $x$ is positive, find the minimum value of $\\frac{\\sqrt{x^{4}+x^{2}+2 x+1}+\\sqrt{x^{4}-2 x^{3}+5 x^{2}-4 x+1}}{x}$.",
        "answer": "\\sqrt{10}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 153551,
        "difficulty": null
      },
      {
        "id": "32b5dd278289c66d4aa03ef244c4ae770d85b65fb65a245e3d360ad9f624ea85",
        "domain": "numina",
        "question": "5-1. A square with a side of 100 was cut into two equal rectangles. They were placed next to each other as shown in the figure. Find the perimeter of the resulting figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_535cfcd84333341737d0g-04.jpg?height=277&width=594&top_left_y=684&top_left_x=731)",
        "answer": "500",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17125,
        "difficulty": null
      },
      {
        "id": "bd2c46d071429dc3a72969e4bfd02778fc66b1611d032f606423188efc5af33b",
        "domain": "numina",
        "question": "Let * be a natural number $n$ a three-digit number, the sum of all three-digit numbers formed by the arbitrary permutation of its three non-zero digits minus $n$ equals 1990. Find $n$.",
        "answer": "452",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94579,
        "difficulty": null
      },
      {
        "id": "f1f4a49434220c3aa8a7de195ccf24774be84dd03002c8da6ac3d6e148213b5f",
        "domain": "numina",
        "question": "Given a fixed triangle $\\triangle A B C$ and a point $P$, find the maximum value of\n\n$$\n\\frac{A B^{2}+B C^{2}+C A^{2}}{P A^{2}+P B^{2}+P C^{2}}\n$$",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 90546,
        "difficulty": null
      },
      {
        "id": "d95c075e0d23abd4962663bbe809931416356d4d2358b9b3414c45fbcac9ff03",
        "domain": "numina",
        "question": "2. Compute\n\n$$\n\\int_{0}^{\\pi / 2} \\cos ^{2}(\\cos x)+\\sin ^{2}(\\sin x) d x\n$$",
        "answer": "\\frac{\\pi}{2}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 25837,
        "difficulty": null
      },
      {
        "id": "6653c21d6fb9ade87b01570a9deef260cce7b98386e478230f99e3495ca0e535",
        "domain": "numina",
        "question": "2. Plane $\\alpha // \\beta, AB=6 \\text{ cm}$ is a fixed line segment between $\\alpha$ and $\\beta$, $CD$ is a moving line segment between $\\alpha$ and $\\beta$, $CD \\perp AB$. Then when the angle between $AB$ and $\\beta$ is $60^{\\circ}$, the minimum value of $CD$ is $\\qquad$",
        "answer": "6\\sqrt{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 119265,
        "difficulty": null
      },
      {
        "id": "3f39b06a5e34c9233f53223b4885774f3a7f698056fc726bae10724bd763754d",
        "domain": "numina",
        "question": "When the last two digits of the square of a natural number are swapped, we get the square of the number that is one greater. Determine all such numbers.\n\nHatározzuk meg az összes ilyen számot.\n\n(Note: The last sentence is kept in Hungarian as it was in the original text, as it seems to be a repetition or a specific instruction in the original context.)",
        "answer": "13",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 171667,
        "difficulty": null
      },
      {
        "id": "a22f048990e7ab3b605a025db1ead844ecce548db31da1fb01527383197aa095",
        "domain": "numina",
        "question": "Example 18 (1993 National High School League Question) Real numbers $x, y$ satisfy $4 x^{2}-5 x y+4 y^{2}=5$. If $s=$ $x^{2}+y^{2}$, then $\\frac{1}{s_{\\max }}+\\frac{1}{s_{\\text {min }}}=$",
        "answer": "\\frac{8}{5}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 131768,
        "difficulty": null
      },
      {
        "id": "9f08ec3b48dd748cbe812ce7c0a20c11bd4198efac510d3ce6a0a28e1467ed29",
        "domain": "numina",
        "question": "Which is the two-digit number that is 1 less than twice the number obtained by swapping its digits? (H)",
        "answer": "73",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 176420,
        "difficulty": null
      },
      {
        "id": "ecbeb74be61e85061117a2f9df1160bbf17aea977e37543a88df2d7a079968cd",
        "domain": "numina",
        "question": "Task A-3.1. (8 points)\n\nSolve the equation $\\quad 4^{\\log x}-32+x^{\\log 4}=0$.",
        "answer": "100",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89385,
        "difficulty": null
      },
      {
        "id": "7e420fa1856570753979700568b55cc16995cb54b5057b9525de3b05738cc9fc",
        "domain": "numina",
        "question": "## 8. The Emir and His Chauffeur\n\nEmir Ben Sidi Mohammed always drives from his palace to the airport at the same speed along a magnificent highway that runs through the desert. If his chauffeur increased the average speed of the car by $20 \\mathrm{km} /$ h, the Emir would save 2 minutes, and if the chauffeur drove at a speed 20 km/h slower, the Emir would lose 3 minutes.\n\nHow far is the Emir's palace from the airport?",
        "answer": "20",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 42064,
        "difficulty": null
      },
      {
        "id": "112ba73d4791376ef7f09fc18910bceecccec6a4f4a3ca9e38e67dc9a033d434",
        "domain": "numina",
        "question": "26. [14] In how many ways can 13 bishops be placed on an $8 \\times 8$ chessboard such that (i) a bishop is placed on the second square in the second row, (ii) at most one bishop is placed on each square, (iii) no bishop is placed on the same diagonal as another bishop, and (iv) every diagonal contains a bishop? (For the purposes of this problem, consider all diagonals of the chessboard to be diagonals, not just the main diagonals).",
        "answer": "1152",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 151345,
        "difficulty": null
      },
      {
        "id": "b3d41392ddc64843960b87a6c6f2fd37c34c2c711e36cbec33662e5a67f2b35d",
        "domain": "numina",
        "question": "2A. A saleswoman needs to receive 1100 gift boxes. In the supply warehouse, there are packages of 70, 40, and 25 gift boxes. The transportation cost for each package is 20, 10, and 7 denars, respectively. Which packages and in what quantities should the saleswoman order to minimize transportation costs? (Packages must not be opened in the warehouse.)",
        "answer": "25",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 190765,
        "difficulty": null
      },
      {
        "id": "c6847636abd69b857ac7e93d7ea07073c2dfce86122ae014be2bba5bfc30544b",
        "domain": "numina",
        "question": "6. Determine the largest constant $K \\in \\mathbb{R}$ with the following property:\n\nIf $a_{1}, a_{2}, a_{3}, a_{4}>0$ are such that for any $i, j, k \\in \\mathbb{N}, 1 \\leq i<j<k \\leq 4$ it holds that $a_{i}^{2}+a_{j}^{2}+a_{k}^{2} \\geq 2\\left(a_{i} a_{j}+a_{j} a_{k}+a_{k} a_{i}\\right)$, then\n\n$$\na_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2} \\geq K\\left(a_{1} a_{2}+a_{1} a_{3}+a_{1} a_{4}+a_{2} a_{3}+a_{2} a_{4}+a_{3} a_{4}\\right)\n$$\n\n(Dušan Djukić)\n\nTime allowed: 270 minutes.\n\nEach problem is worth 7 points.",
        "answer": "\\frac{11}{7}",
        "topic": "Inequalities",
        "source": "olympiads",
        "source_index": 162461,
        "difficulty": null
      },
      {
        "id": "62acf4c09667b70bbffd774b8521e7b1f6fc2c975187fe663ed34aae70d9219b",
        "domain": "numina",
        "question": "29. The robot is on a rectangular field that is 16 meters long and 8 meters wide, walking along the paths indicated by the arrows for a performance. How many meters does the robot travel when it goes from point $A$ to point $B$? Assume that the width between adjacent parallel paths in the figure is 1 meter (the distance from point $B$ to the nearest vertical path is also 1 meter)",
        "answer": "150",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 99335,
        "difficulty": null
      },
      {
        "id": "993283ca5c2369b96ac6e828703c98dd5bdfdcdee49b47fe3ea8dc09c4cbb1a6",
        "domain": "numina",
        "question": "[ Divisibility of numbers. General properties ] $[$ Classical combinatorics (other) $]$\n\nHow many natural numbers less than a thousand are not divisible by either 5 or 7?",
        "answer": "686",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 57353,
        "difficulty": null
      },
      {
        "id": "0984196186da69d8fd2ca4b07743baf833a953e425e17e331aa44c05ddc73546",
        "domain": "numina",
        "question": "Task 1 - 120521\n\\(4 * * \\cdot 3 * *\\)\n    \\(* * * 5\\)\n        \\(3 * * *\\)\n            \\(8 * *\\)\n                            In the following task, each asterisk \\(\\left({ }^{*}\\right)\\) is to be replaced by one of the digits\n    \\(* * * * 3 *\\)\n                            \\(0,1,2,3,4,5,6,7,8,9\\) so that a correctly solved multiplication\n                            problem is formed.\nEach line must begin with a non-zero digit.\nAs a result, only a correctly completed problem is required without any justification.",
        "answer": "158530",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 182992,
        "difficulty": null
      },
      {
        "id": "5dab812c75c8f69e713b2cfc6bc152e017d3cb3efa6f60139d9d96a4853f0809",
        "domain": "numina",
        "question": "4. Problem: Determine all prime numbers $p$ such that $p^{2}-6$ and $p^{2}+6$ are both prime numbers.",
        "answer": "5",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 153980,
        "difficulty": null
      },
      {
        "id": "f13281f3d68962320e99283d215129af8f221f45c05d6eb88d79b5bebb99d53a",
        "domain": "numina",
        "question": "5. A part is machined from a metal blank. The chips obtained from machining 8 parts can be remelted into one blank. How many parts can be made from 64 blanks?",
        "answer": "73",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 42654,
        "difficulty": null
      },
      {
        "id": "3d48b9400a3219f79c49b7fa18cf4c4fcb59a2fadb5319d05dd6202ec1662a5d",
        "domain": "numina",
        "question": "9. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then the sum $f\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+f\\left(\\frac{3}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)$ equals $\\qquad$ .",
        "answer": "500",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104665,
        "difficulty": null
      },
      {
        "id": "359f382aba418cbd1db1acd98a9150de6e02bff1f9ee590899e5b6c54e1c8d70",
        "domain": "numina",
        "question": "Task B-1.2. Using the digits 1, 3, 4, 5, and a, where $a$ is also some digit (not necessarily distinct), write the largest possible five-digit number that is divisible by 12. Each of the digits $1,3,4,5, a$ must be used.",
        "answer": "54312",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 87679,
        "difficulty": null
      },
      {
        "id": "7d490377366d23fab73ad6c165b5d544228b61f7ac7c89eee08f5cde966efeee",
        "domain": "numina",
        "question": "3. One working day, $40\\%$ of the students from the first shift in a primary school bought a bagel for lunch, $36\\%$ of the students bought a sandwich, and the rest bought a croissant.\n\nIn the second shift at the same school:\n\n- the number of students who bought a croissant for lunch is $37.5\\%$ higher than the number of students from the same school in the first shift who bought a croissant for lunch;\n- the number of students who bought a sandwich for lunch is $75\\%$ higher than the number of students from the same school in the first shift who bought a sandwich for lunch; and\n- the number of students who bought a bagel for lunch is $75\\%$ lower than the number of students from the same school in the first shift who bought a bagel for lunch.\n\nIn which shift is the number of students greater than the number of students in the other shift? By what percentage is it greater?",
        "answer": "6",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 189093,
        "difficulty": null
      },
      {
        "id": "542f9c6dad8ed845734bec936e068f07db63721aaf87b98ebc0ed009f048e700",
        "domain": "numina",
        "question": "B5. Simon has 2017 blue blocks numbered from 1 to 2017. He also has 2017 yellow blocks numbered from 1 to 2017. Simon wants to arrange his 4034 blocks in a row. He wants to do so in such a way that for every $k=1,2, \\ldots, 2017$ the following conditions are met:\n\n- to the left of the blue block with number $k$ there are $k$ or more yellow blocks;\n- to the right of the yellow block with number $k$ there are $k$ or fewer blue blocks.\n\nDetermine all possible numbers of the 1000th block from the left in the row.\n\n## C-problems\n\nFor the C-problems, not only the answer is important; your reasoning and the way of solving it must also be clearly written down. Solve each C-problem on a separate sheet of paper. Each correctly worked out C-problem is worth 10 points. Partial solutions can also earn points. Therefore, write everything clearly and submit your scratch paper (for each problem!) as well.",
        "answer": "500",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 85575,
        "difficulty": null
      },
      {
        "id": "a62ddb7a26730029df3d6ef4534954ea183ce6f10723c954b3bb30f25bb69587",
        "domain": "numina",
        "question": "3. In a row, all natural numbers less than a billion that have exactly 13 natural divisors (including one and the number itself) were written down. How many of them have an even sum of digits?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 43300,
        "difficulty": null
      },
      {
        "id": "52247bc39b1642a1ff6e3c0c16f42b1e32c4e5419f29210545b394fb7a7ce8b2",
        "domain": "numina",
        "question": "XXI OM - I - Problem 12\n\nFind the smallest positive number $ r $ with the property that in a regular tetrahedron with edge length 1, there exist four points such that the distance from any point of the tetrahedron to one of them is $ \\leq r $.",
        "answer": "\\frac{\\sqrt{6}}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 1239,
        "difficulty": null
      },
      {
        "id": "77af455fd54269a93927dd10cb64d44b6aa8cc84023b1acb144e8cb6b2d371af",
        "domain": "numina",
        "question": "Find the smallest positive integer $n$ with the following property: There does not exist an arithmetic progression of 1993 terms of real numbers containing exactly $n$ integers.",
        "answer": "70",
        "topic": "Number Theory",
        "source": "olympiads_ref",
        "source_index": 194509,
        "difficulty": null
      },
      {
        "id": "78c3129966728c5275ab2724bc4f0ab7ef48b6c1d3a8704db90c1a032b6fa4d5",
        "domain": "numina",
        "question": "44. On the Euclidean plane are given 14 points:\n$$\n\\begin{array}{l}\nA=(0,428) \\quad B=(9,85) \\quad C=(42,865) \\quad D=(192,875) \\\\\nE=(193,219) \\quad F=(204,108) \\quad G=(292,219) \\quad H=(316,378) \\\\\nI=(375,688) \\quad J=(597,498) \\quad K=(679,766) \\quad L=(739,641) \\\\\nM=(772,307) \\quad N=(793,0) \\\\\n\\end{array}\n$$\n\nA fly starts at $A$, visits all the other points, and comes back to $A$ in such a way as to minimize the total distance covered. What path did the fly take? Give the names of the points it visits in order. Your score will be\n$$\n20+\\lfloor\\text { the optimal distance }\\rfloor-\\lfloor\\text { your distance }\\rfloor\n$$\nor 0 , whichever is greater.",
        "answer": "3591.22",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 152421,
        "difficulty": null
      },
      {
        "id": "6d97102b2e71f7166d21c092fab6a146b361362cd38ab7453c7852db46775e0c",
        "domain": "numina",
        "question": "3. The product of the areas of the six faces of a rectangular prism is 14641, then the volume of the rectangular prism is\n\n保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "11",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 107907,
        "difficulty": null
      },
      {
        "id": "bd63581aacbb746d90c13cce55b8820d8743b60230b47b22f910d0f5752fac1c",
        "domain": "numina",
        "question": "1. Positive integers $a, b$, and $c$ are all powers of $k$ for some positive integer $k$. It is known that the equation $a x^{2}-b x+c=0$ has exactly one real solution $r$, and this value $r$ is less than 100. Compute the maximum possible value of $r$.",
        "answer": "64",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 151916,
        "difficulty": null
      },
      {
        "id": "cb0eddd2794e7215708bf3c979cf82c748965032ca1afa099f746fc0c5433bf2",
        "domain": "numina",
        "question": "663. Find all natural $n$ for which the number $n^{2}+3 n$ is a perfect square.",
        "answer": "1",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 45236,
        "difficulty": null
      },
      {
        "id": "f93e3f02c5c1dc6116d352f32bed4b6a11edfb5adbb64b22c78d23e413f2718c",
        "domain": "numina",
        "question": "## Task B-1.7.\n\nFrom point $D$ on side $\\overline{A B}$, where $|A D|>|B D|$, of an equilateral triangle $A B C$, a perpendicular is drawn to side $\\overline{B C}$ with its foot at $E$. Then, from $E$, a perpendicular is drawn to $\\overline{C A}$ with its foot at $F$, and from $F$, a perpendicular is drawn to $\\overline{A B}$ with its foot at $G$. The area of quadrilateral $D E F G$ is $21 \\sqrt{3} \\mathrm{~cm}^{2}$, and the length of the side of triangle $A B C$ is $14 \\mathrm{~cm}$. Determine the length $x=|B D|$ and the perimeter of quadrilateral $D E F G$.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 89540,
        "difficulty": null
      },
      {
        "id": "c94df77fd86ecb1a7154f2d594759fa7a97e884ce68a376be0724cec3414f3e7",
        "domain": "numina",
        "question": "4. Solve the system of equations:\n\n$$\n\\begin{aligned}\n\\lfloor 2 x\\rfloor+\\lfloor x\\rfloor+\\lfloor 7 x\\rfloor & =2017 \\\\\n3\\{x\\}\\{5 x\\} & =4\\{2 x\\}\\{6 x\\}\n\\end{aligned}\n$$\n\n(With $\\lfloor x\\rfloor$ we denote the greatest integer not greater than $x$, and with $\\{x\\}$ we denote $x-\\lfloor x\\rfloor$.)",
        "answer": "\\frac{2221}{11}",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 161402,
        "difficulty": null
      },
      {
        "id": "ee0ae03be440d0bdf18620feb495704983d1321e16fc9d287cd130910dea95d9",
        "domain": "numina",
        "question": "Kazitsyna T.v.\n\nFour mice: White, Gray, Fat, and Thin were dividing a piece of cheese. They cut it into 4 visually identical slices. Some slices had more holes, so Thin's slice weighed 20 grams less than Fat's, and White's slice weighed 8 grams less than Gray's. However, White was not upset because his slice weighed exactly one quarter of the total weight of the cheese.\n\nGray cut off 8 grams from his piece, and Fat cut off 20 grams. How should the mice divide the resulting 28 grams of cheese so that everyone has an equal amount of cheese? Don't forget to explain your answer.",
        "answer": "14",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 56586,
        "difficulty": null
      },
      {
        "id": "5ff8c58d2a8d61261813624e0715144464be443c791643d6037940c01de5f15f",
        "domain": "numina",
        "question": "The price of gas was raised three times in the first half of the year, by $5\\%$, $6\\%$, and $10\\%$ respectively. Analysts predicted that the gas price would increase by a total of 1/3 over the course of the year. According to this, what percentage increase in the price of gas can be expected for the rest of the year?",
        "answer": "8.91",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 171806,
        "difficulty": null
      },
      {
        "id": "3ec016c214ade7571f9a4ee5d191bcdb2439cd058b29be59ea84bcd61b6039f8",
        "domain": "numina",
        "question": "What fraction of the Earth's volume lies above the 45 degrees north parallel? You may assume the Earth is a perfect sphere. The volume in question is the smaller piece that we would get if the sphere were sliced into two pieces by a plane.",
        "answer": "\\frac{8-5\\sqrt{2}}{16}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 151735,
        "difficulty": null
      },
      {
        "id": "d9d840daec8b267e933f1ac0060366a78965c632ab456ddecf9759b361d3f416",
        "domain": "numina",
        "question": "# Problem №3\n\nA New Year's garland hanging along the school corridor consists of red and blue bulbs. Next to each red bulb, there is definitely a blue one. What is the maximum number of red bulbs that can be in this garland if there are 50 bulbs in total?",
        "answer": "33",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17544,
        "difficulty": null
      },
      {
        "id": "3fcef6a9631b1a9feea722069ba60f59b58f924a5b45339d03863f81ccd6eb86",
        "domain": "numina",
        "question": "75. Arrange 10 small balls labeled with numbers 1 to 10 in a row, such that the sum of the numbers on every 3 adjacent balls is a multiple of 3. The total number of arrangements is $\\qquad$.",
        "answer": "1728",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 134117,
        "difficulty": null
      },
      {
        "id": "209cbb09581b9473fd9892421f6b44d2f97cdf4866959a224564dd3acebe0e6b",
        "domain": "numina",
        "question": "Example 1 (to $4^{\\circ}$). Find $y^{\\prime}$ if $10 x^{3}+4 x^{2} y+y^{2}=0$. Also find $y^{\\prime}$ at $x=-2, y=4$.",
        "answer": "-\\frac{7}{3}",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 30246,
        "difficulty": null
      },
      {
        "id": "62c4a51cd531a2c95751fc7eaae7a1c93fe12adc88db01afd99d7391960239ec",
        "domain": "numina",
        "question": "7. (8 points) In the equation $\\frac{2 \\frac{3}{5}-(8.5-\\square) \\div 3.5}{1 \\div\\left(3.05+4 \\frac{9}{20}\\right)}=2$, $\\square$ represents a number. Therefore, $\\square=$",
        "answer": "\\frac{1}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 93162,
        "difficulty": null
      },
      {
        "id": "07fb1a3340a27753824f2233060a8f0c109e4f59617e4dc33d03dcd6b50df9dc",
        "domain": "numina",
        "question": "9.6. Each of the 1000 gnomes has a hat, blue on the outside and red on the inside (or vice versa). If a gnome is wearing a red hat, he can only lie, and if it is blue, he can only tell the truth. During one day, each gnome said to each other, \"You have a red hat!\" (during the day, some gnomes turned their hats inside out). Find the minimum possible number of turnings. (I. Bogdanov)",
        "answer": "998",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 13572,
        "difficulty": null
      },
      {
        "id": "857fa4dd3809c4646723c3ccc295649020909120f19b429c2346ce32a46ec029",
        "domain": "numina",
        "question": "Example 7 Let $S=\\{1,2, \\cdots, 2005\\}$. If any set of $n$ pairwise coprime numbers in $S$ contains at least one prime number, find the minimum value of $n$.",
        "answer": "16",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 96941,
        "difficulty": null
      },
      {
        "id": "7843e1baa4858d505f930169d7b6388f2d4437aad3ba294b6d41257c746ee4df",
        "domain": "numina",
        "question": "## 3. Hussar\n\nThe captain of a hussar ship divides gold coins with his thirteen hussars in the following way:\n\nfirst division: one for me, one for each of you second division: two for me, one for each of you third division: three for me, one for each of you.\n\nIn each subsequent division, the captain takes one more gold coin than in the previous division, and gives each of the other hussars one gold coin.\n\nHow many divisions are needed for the captain to have exactly 187500 more gold coins than all his hussars combined?\n\n## Result: $\\quad 625$",
        "answer": "625",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 89896,
        "difficulty": null
      },
      {
        "id": "f80d6f16ae1bbdd0888cceb469a5111bb90f0969a2ac06ea0a06078ce5dba2cc",
        "domain": "numina",
        "question": "Problem 130. Given an angle of $54^{\\circ}$. Divide it into three equal parts using a compass and a straightedge.",
        "answer": "18",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 39849,
        "difficulty": null
      },
      {
        "id": "a10c5c07b2971a278a9dfe565464e65fa7e3ab1294117551011f493a021c2171",
        "domain": "numina",
        "question": "The first two elements of the sequence $x_{n}$ are $x_{1}=1001, x_{2}=1003$ and if $n \\geq 1$, then $x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}$. What is the sum of the first 2004 elements of the sequence?",
        "answer": "1338004",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 170695,
        "difficulty": null
      },
      {
        "id": "a77fa01dacc3a5fd8c1bd39fcb8940c211238869dfdb30462e89e3bc69efd601",
        "domain": "numina",
        "question": "8.3. In triangle $A B C$, side $A C$ is the largest. Points $M$ and $N$ on side $A C$ are such that $A M=A B$ and $C N=C B$. It is known that angle $N B M$ is three times smaller than angle $A B C$. Find $\\angle A B C$.",
        "answer": "108",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 15345,
        "difficulty": null
      },
      {
        "id": "6f53a5ef92fa0f0d5b11d7125362121d063e527a3e6e839272a8b03f4ea9ebcf",
        "domain": "numina",
        "question": "## Task 2 - 340712\n\nFor a triangle $A B C$, it is required: The angle bisector through $A$ and the perpendicular bisector of $A B$ intersect at a point $D$, which lies on the side $B C$.\n\na) What size must the angle $\\angle A C B$ have in a triangle that meets this requirement and in which the angle $\\angle A B C$ has a size of $35^{\\circ}$? Draw such a triangle!\n\nb) Draw a triangle that also meets the above requirement and in which the angle $\\angle A B C$ has a size of $50^{\\circ}$!\n\nc) Is there a triangle that also meets the above requirement and in which the angle $\\angle A B C$ has a size of $60^{\\circ}$?",
        "answer": "75",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 183876,
        "difficulty": null
      },
      {
        "id": "4ea9d16ad59945862c11e250384b75a456db87dadc7ce0f4fc6ef860e5b0510e",
        "domain": "numina",
        "question": "Example 4 Find the maximum value of the function $f(x)=\\sqrt{x^{4}-3 x^{2}-6 x+13}-\\sqrt{x^{4}-x^{2}+1}$.",
        "answer": "\\sqrt{10}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 94361,
        "difficulty": null
      },
      {
        "id": "cfbd87e55be96ec22de61d8558b421e99720ac4123422d388a8780ca5ceda03d",
        "domain": "numina",
        "question": "8. For any real number $x$, $[x]$ represents the greatest integer not exceeding $x$. When $0 \\leqslant x \\leqslant 100$, the range of $f(x)=[3 x]+[4 x]+[5 x]+[6 x]$ contains $\\qquad$ different integers.",
        "answer": "1201",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 121563,
        "difficulty": null
      },
      {
        "id": "4103aa1aac20c4b357e3e6f672aeba40e5eee782e01702b066f4d6941cccd909",
        "domain": "numina",
        "question": "5. (10 points) Five identical non-ideal ammeters are connected as shown in the figure. Ideal power supply is connected to points $A$ and $B$. Determine the sum of the readings of all ammeters, given that the reading of the first ammeter $I_{1}=2 \\mathrm{MA}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_9c0cded10981f14efe49g-03.jpg?height=774&width=1108&top_left_y=458&top_left_x=494)",
        "answer": "24",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 3964,
        "difficulty": null
      },
      {
        "id": "65ea45fe6d936bd8cbdceec18f2d9c17347840c1a20042136249f308c0588682",
        "domain": "numina",
        "question": "$3.462 A=\\operatorname{tg}\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arctg}(-2)\\right)$\n\n$3.462 A=\\tan\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\arctan(-2)\\right)$",
        "answer": "-\\frac{1}{2}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 51893,
        "difficulty": null
      },
      {
        "id": "38871d2068182ddb451bda945038674316a172bebdf22c002d1222ae1c4293e7",
        "domain": "numina",
        "question": "2. Let $A B C D A' B' C' D'$ be a cube with edge $a$, where $A B C D$ is one of the faces of the cube, and $A A', B B', C C'$, and $D D'$ are edges of the cube. Calculate (in terms of $a$) the height of the pyramid $A C B' D'$, dropped from the vertex $D$.",
        "answer": "\\frac{2\\sqrt{3}}{3}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 190610,
        "difficulty": null
      },
      {
        "id": "6a88c797c1d6f124edccec90c39a182fd2ce3151b3a02a59d63e4ffef05d8a1f",
        "domain": "numina",
        "question": "Problem 9.5. On the base $AC$ of isosceles triangle $ABC (AB = BC)$, a point $M$ is marked. It is known that $AM = 7, MB = 3, \\angle BMC = 60^\\circ$. Find the length of segment $AC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_8af0c885427e3e323cf9g-33.jpg?height=240&width=711&top_left_y=86&top_left_x=369)",
        "answer": "17",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12120,
        "difficulty": null
      },
      {
        "id": "bbdebff038058882f0939f61c3de0a2e2473aae62526b339a5397aac9a7264a0",
        "domain": "numina",
        "question": "Question 189: In a unit square, if two of its four sides are colored with color $i$, and the other two sides are colored with two different colors other than $i$, then the unit square is called “dominated by color $i$”. As shown in the figure, a $1 \\times 3$ grid table contains a total of 10 unit line segments. Now, these 10 line segments are to be colored, each with one of the three colors: red, yellow, or blue, such that there is one unit square dominated by red, one by yellow, and one by blue. The number of such coloring methods is $\\qquad$ _.",
        "answer": "1026",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 97807,
        "difficulty": null
      },
      {
        "id": "4f83226cbea10f9a1aefaa17114fb326bc3df38ed47e9dbdbed0ec552d1d7dae",
        "domain": "numina",
        "question": "$\\left[\\begin{array}{l}\\text { Word Problems (miscellaneous) }) \\\\ {[\\text { Common Fractions }}\\end{array}\\right]$\n\nA flock of geese was flying, and a single goose was flying towards them and said: \"Greetings, a hundred geese!\" The leader of the flock replied to him: \"No, we are not a hundred geese! You see, if there were as many of us as there are, and as many again, and half as many, and a quarter as many, and you, goose, with us, then we would be a hundred geese, but as it is...\" How many geese were in the flock?",
        "answer": "36",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 72255,
        "difficulty": null
      },
      {
        "id": "138001c5c963a63ef3f57a24115134e8b7c9e8a8a6bbd1d4e1e2084f344c3f52",
        "domain": "numina",
        "question": "Problem 6.8. There are exactly 120 ways to color five cells in a $5 \\times 5$ table so that each column and each row contains exactly one colored cell.\n\nThere are exactly 96 ways to color five cells in a $5 \\times 5$ table without a corner cell so that each column and each row contains exactly one colored cell.\n\nHow many ways are there to color five cells in a $5 \\times 5$ table without two corner cells so that each column and each row contains exactly one colored cell?\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-18.jpg?height=258&width=1008&top_left_y=530&top_left_x=220)",
        "answer": "78",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 18110,
        "difficulty": null
      },
      {
        "id": "f31f0eddee4969e599648e0dc77c27272529231ea9096fe2e1e6a8528ea043dc",
        "domain": "numina",
        "question": "## Problem 4\n\nKing George has decided to connect the 1680 islands in his kingdom by bridges. Unfortunately the rebel movement will destroy two bridges after all the bridges have been built, but not two bridges from the same island.\n\nWhat is the minimal number of bridges the King has to build in order to make sure that it is still possible to travel by bridges between any two of the 1680 islands after the rebel movement has destroyed two bridges?",
        "answer": "2016",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 460,
        "difficulty": null
      },
      {
        "id": "48f584dd55f3de2ce28d1f8c7d74718d1e06c17dbff7fe7321fa52656516aad0",
        "domain": "numina",
        "question": "Problem 2. In a football tournament, six teams played: each team played once with each other. In the next round, teams that scored twelve or more points qualify. Three points are awarded for a win, one point for a draw, and zero points for a loss. What is the maximum number of teams that can advance to the next round?",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 9127,
        "difficulty": null
      },
      {
        "id": "00d19cac8d0938066a70b09bcde674573819f0ab4e6f20482a70192bc7008ac2",
        "domain": "numina",
        "question": "94. Fill in 9 different natural numbers in a 3x3 grid, satisfying: in each row, the sum of the two left numbers equals the rightmost number; in each column, the sum of the two upper numbers equals the bottom number. What is the smallest number that can be in the bottom-right corner? $\\qquad$",
        "answer": "12",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 106406,
        "difficulty": null
      },
      {
        "id": "daa5e64a61176ac8c4b04152de23472e310c98c90e25d038f9ecf9891c5ad705",
        "domain": "numina",
        "question": "3. Let $a$, $b$, $c$ be the roots of\n$$\nf(x)=x^{3}+3 x^{2}-10 x+5\n$$\n\nThen $a^{2} b^{2}+b^{2} c^{2}+c^{2} a^{2}=$ $\\qquad$",
        "answer": "70",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 119602,
        "difficulty": null
      },
      {
        "id": "beb1a3fa6f36106b4c65c641d2b7f338c8ceed68b34ff320cc3b2d433337790f",
        "domain": "numina",
        "question": "11. Given two lines $I_{1}: 3 x+4 y-25=0, I_{2}: 117 x-44 y-175=0$, point $A$ has projections $B, C$ on lines $I_{1}, l_{2}$, respectively. (1) Find the locus curve $\\Gamma$ of point $A$ such that $S_{\\triangle A B C}=\\frac{1728}{625}$; if circle $T:\\left(x-\\frac{39}{5}\\right)^{2}+\\left(y-\\frac{27}{5}\\right)^{2}=r^{2}(r>0)$ intersects curve $\\Gamma$ at exactly 7 points, find the value of $r$.",
        "answer": "8",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 114774,
        "difficulty": null
      },
      {
        "id": "89ebd0267c728ed72021f2b441046daa09bd6d4a19474a2445595eccb3f2f751",
        "domain": "numina",
        "question": "9. (10 points) The area of rectangle $A B C D$ is 416 square centimeters, and the vertex $F$ of trapezoid $A F G E$ is on $B C$, with $D$ being the midpoint of the waist $E G$. Try to find the area of trapezoid $A F G E$.",
        "answer": "416",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 113353,
        "difficulty": null
      },
      {
        "id": "4e9308fbaf3e126833e1220e68e9f03923bfb114c9d0500fc1a4c1d98f03f68f",
        "domain": "numina",
        "question": "8. In $\\triangle A B C$, $a, b, c$ are the sides opposite to angles $A, B, C$ respectively. Given $a+c=2 b, A-C=\\frac{\\pi}{3}, \\sin B=$ $\\qquad$ .",
        "answer": "\\frac{\\sqrt{39}}{8}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 133331,
        "difficulty": null
      },
      {
        "id": "7cf69695f7d8a96c98a16d4fbbfe873feb5d91708abf325c26b1fea9303f5d11",
        "domain": "numina",
        "question": "3. Let real numbers $x, y$ satisfy\n$$\nx^{2}+\\sqrt{3} y=4, y^{2}+\\sqrt{3} x=4, x \\neq y \\text {. }\n$$\n\nThen the value of $\\frac{y}{x}+\\frac{x}{y}$ is . $\\qquad$",
        "answer": "-5",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 104739,
        "difficulty": null
      },
      {
        "id": "594b40de2a3a90217c25c3084d9829b58068430b82b94da5e73453ffb35d874f",
        "domain": "numina",
        "question": "8. Let $z_{1} 、 z_{2}$ be complex numbers, and satisfy:\n$$\n\\left|z_{1}\\right|=5, \\frac{z_{1}}{z_{2}}=2+\\mathrm{i} \\text {. }\n$$\n\nThen $\\left|z_{1}-z_{2}\\right|=$",
        "answer": "\\sqrt{10}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 97978,
        "difficulty": null
      },
      {
        "id": "2e93af31f947d98326b03cfde5887c903621fd84365a45d02247ebf6d73a27ca",
        "domain": "numina",
        "question": "8.21 Two players, A and B, take turns removing stones from a pile of $n$ stones. A starts first, and on his first turn, he can take any number of stones but not all and not zero. Thereafter, each player must take a number of stones that is a divisor of the number of stones the other player took on their last turn. The player who takes the last stone wins. For what smallest $n>1992$ does B have a winning strategy?",
        "answer": "2048",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 146162,
        "difficulty": null
      },
      {
        "id": "45c827d3593672cec86f64bfae5ce8e65983cbcdb38260ade29d59a506c5863c",
        "domain": "numina",
        "question": "8. The numbers $1059, 1417, 2312$ leave the same remainder when divided by $d$, $d>1$, then the integer $d=$\n\n保留了源文本的换行和格式。",
        "answer": "179",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 103901,
        "difficulty": null
      },
      {
        "id": "ee5af8856ff4b2a6e472e5c3720db027a22f249f9abf3507d946656d97055753",
        "domain": "numina",
        "question": "Problem 9.7. In triangle $ABC$, the bisector $AL$ is drawn. Points $E$ and $D$ are marked on segments $AB$ and $BL$ respectively such that $DL = LC$, $ED \\parallel AC$. Find the length of segment $ED$, given that $AE = 15$, $AC = 12$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d7ec6e60d14a8fdbf750g-37.jpg?height=254&width=410&top_left_y=91&top_left_x=526)",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17177,
        "difficulty": null
      },
      {
        "id": "8bddf322b358fc7295bfe28eb3bec92dd19d6354012f95ec3deb7a096c41d255",
        "domain": "numina",
        "question": "14. (6 points) Two cars, A and B, start from points $A$ and $B$ respectively and travel towards each other, meeting at a point 108 kilometers from $B$. After reaching each other's starting point, they immediately return to their original locations, and meet again at a point 84 kilometers from $A$. The distance between the two meeting points is $\\qquad$ kilometers.",
        "answer": "48",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 129881,
        "difficulty": null
      },
      {
        "id": "13bd91db1206d888e73e8bc7b27d0e3afd11b82fa3e7e1101ce9215657c61c42",
        "domain": "numina",
        "question": "I2.1 Suppose $P$ is an integer and $5<P<20$. If the roots of the equation $x^{2}-2(2 P-3) x+4 P^{2}-14 P+8=0$ are integers, find the value of $P$.",
        "answer": "12",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 152957,
        "difficulty": null
      },
      {
        "id": "e0f2711fcb1cffa47bb8bff1da3708fe45030d4a6962452dbfa73bf1fe9c7c2a",
        "domain": "numina",
        "question": "Problem 11.3. There are 2001 towns in a country every one of which is connected with at least 1600 towns by direct bus line. Find the largest $n$ for which there exist $n$ towns any two of which are connected by direct bus line.",
        "answer": "5",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 177315,
        "difficulty": null
      },
      {
        "id": "07afc98c41dcc471c60cc8dcf86cf5fceacbba0a72960ec230f6ce55029893f8",
        "domain": "numina",
        "question": "Problem 3. In a warehouse, there are 360 boxes of apples and 384 boxes of pears. The boxes of fruit need to be delivered to different stores in shipments where there is always the same number of fruit boxes and in each shipment, there is the same number of apple boxes. The trucks used to transport the fruit shipments can carry a maximum of 60 boxes. How many boxes of apples and how many boxes of pears are there in each shipment?",
        "answer": "15",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 188895,
        "difficulty": null
      },
      {
        "id": "0341cecf7ef8cdf3157480d692de66f8c2f1088afa1f6ac76188f85e84893d97",
        "domain": "numina",
        "question": "Khachaturyan A.V.\n\nSide $AB$ of triangle $ABC$ is divided into $n$ equal parts (division points $B_{0}=A, B_{1}, B_{2}, B_{n}=B$), and side $AC$ of this triangle is divided into $n+1$ equal parts (division points $C_{0}=A, C_{1}, C_{2}, \\ldots, C_{n+1}=C$). The triangles $C_{i} B_{i} C_{i+1}$ are shaded. What fraction of the area of the triangle is shaded?",
        "answer": "\\frac{1}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 58510,
        "difficulty": null
      },
      {
        "id": "0d9a6a050c19c9e63a83d51921c141cc2fb4511d933da147a84bfd0487090eb5",
        "domain": "numina",
        "question": "29.3.14 Xuan Yuan seeks the smallest natural number $n$ with the property that any $n$-element subset of $\\{1,2, \\cdots, 60\\}$ must contain three numbers that are pairwise coprime.",
        "answer": "41",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 122748,
        "difficulty": null
      },
      {
        "id": "90d29249bf9c3a1892caa19a56d4602c39d29fb90026cb7ca18fce593bd23f1c",
        "domain": "numina",
        "question": "Problem 8.7.1. Given trapezoid $A B C D (B C \\| A D)$. Point $H$ on side $A B$ is such that $\\angle D H A=$ $90^{\\circ}$. It is known that $C H=C D=13$ and $A D=19$. Find the length of segment $B C$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0c3ddd33c7236a6f213bg-10.jpg?height=412&width=579&top_left_y=309&top_left_x=437)",
        "answer": "9.5",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 12958,
        "difficulty": null
      },
      {
        "id": "690b6b11c43e41c42af66e450b8828c37c9f8575780136605d2bc2900255e93c",
        "domain": "numina",
        "question": "8.2.1. (12 points) The first, second, and third terms of a geometric progression are pairwise distinct and are equal to the second, fourth, and seventh terms of a certain arithmetic progression, respectively, and the product of these three numbers is 64. Find the first term of the geometric progression.",
        "answer": "\\frac{8}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6523,
        "difficulty": null
      },
      {
        "id": "25ffbfa618436f11b29244ed175e5a3926a0993fa81ac56412efb99f074aeb83",
        "domain": "numina",
        "question": "1. We call a positive integer a shuffle number if the following hold:\n\n(1) All digits are nonzero.\n\n(2) The number is divisible by 11 .\n\n(3) The number is divisible by 12. If you put the digits in any other order, you again have a number that is divisible by 12 .\n\nHow many 10-digit shuffle numbers are there?",
        "answer": "2",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 85706,
        "difficulty": null
      },
      {
        "id": "65f4a8fa91253aab4db9b50b3995d375f787be6b462f94c13515823e9db0ebad",
        "domain": "numina",
        "question": "8. $(8,9)$ It is known that the number $\\frac{(2+\\sqrt{3})^{4}+(2-\\sqrt{3})^{4}}{(2+\\sqrt{3})^{6}+(2-\\sqrt{3})^{6}}$ is rational. Write this number as an irreducible fraction.",
        "answer": "\\frac{97}{1351}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 5045,
        "difficulty": null
      },
      {
        "id": "26458880b7527626f52763cd56c36c25757f497b4fca113cb6299a266655f871",
        "domain": "numina",
        "question": "## Task A-3.5.\n\nThe base of the pyramid is a regular $n$-sided polygon. Each side of the base is painted black, while each diagonal of the base and each lateral edge of the pyramid is painted either red or blue. Determine the smallest natural number $n \\geqslant 4$ for which there necessarily exists a triangle whose vertices are vertices of the pyramid and all three sides are of the same color.",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 89625,
        "difficulty": null
      },
      {
        "id": "c30f10565ec8db2134334cbe78711ee1b77ad736d4a7023ab4986d77d396b54c",
        "domain": "numina",
        "question": "In the triangle $ABC$ below, there is a point $D$ on its side $AB$, such that $AB=CD$, $\\widehat{ABC}=100^{\\circ}$, and $\\widehat{DCB}=40^{\\circ}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4fd1b602786ad9981479g-40.jpg?height=455&width=894&top_left_y=478&top_left_x=460)\n\na) What is the measure of the angle $\\widehat{BDC}$?\n\nb) What is the measure of the angle $\\widehat{ACD}$?\n\n#",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73381,
        "difficulty": null
      },
      {
        "id": "7acd837b7c045cd6c1b4f52fe9f999e0767ea2b03847992750fb37446020fbfd",
        "domain": "numina",
        "question": "# Task 10.5\n\nThe altitudes AD and $\\mathrm{BE}$ of an acute-angled triangle $\\mathrm{ABC}$ intersect at point $\\mathrm{H}$.\n\nThe circumcircle of triangle $\\mathrm{ABH}$ intersects sides $\\mathrm{AC}$ and $\\mathrm{BC}$ at points $\\mathrm{F}$ and $\\mathrm{G}$, respectively. Find FG, if $\\mathrm{DE}=5$ cm.\n\n## Number of points 7\n\n#",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 17628,
        "difficulty": null
      },
      {
        "id": "93ff8320bb94f4f5ed4dd09a34f9131b3a0d36c17432c3a76944df9f8977c74c",
        "domain": "numina",
        "question": "D Given that $\\alpha^{2005}+\\beta^{2005}$ can be expressed as a bivariate polynomial in terms of $\\alpha+\\beta$ and $\\alpha \\beta$, find the sum of the coefficients of this polynomial. (Supplied by Zhu Huawei)",
        "answer": "1",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 95824,
        "difficulty": null
      },
      {
        "id": "78d9fd67946a6f2ec3b4b78b3bf3b01ffe6036bc13ca2da2001c215655888ce9",
        "domain": "numina",
        "question": "85. The Fly and the Cars. The length of the road is 300 km. Car $A$ starts at one end of the road at noon and moves at a constant speed of 50 km/h. At the same time, at the other end of the road, car $B$ starts at a constant speed of 100 km/h and a fly, flying at 150 km/h. Upon meeting car $A$, the fly turns and flies towards $B$.\n1) When will the fly meet $B$?\n2) If, upon meeting $B$, the fly turned, flew towards $A$, met him, turned again, and continued flying back and forth between $A$ and $B$ until they collided, when would the cars crush the fly?",
        "answer": "2",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 41001,
        "difficulty": null
      },
      {
        "id": "863008ba1c03dd32054584df7ae1537e53a174cb842a397a2b7e17100ec9b810",
        "domain": "numina",
        "question": "94. There are three cars $A$, $B$, and $C$ that start from the same location at the same time, driving along the same road. They catch up to Xiaolong, who is riding in the same direction, at 5 minutes, 10 minutes, and 12 minutes, respectively. It is known that the speed of $A$ is 24 kilometers/hour, and the speed of $B$ is 20 kilometers/hour. Then the speed of $C$ is $\\qquad$ kilometers/hour.",
        "answer": "\\frac{58}{3}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 103930,
        "difficulty": null
      },
      {
        "id": "2ada74e3086637a91dec816c8687eb6502935dd33dac3ce4deb550e9e35ff61a",
        "domain": "numina",
        "question": "61. Another task about a piece of land\n\n- Here's another task,- said the Black Queen. Another farmer had a piece of land. On one third of his land, he grew pumpkins, on one fourth he planted peas, on one fifth he sowed beans, and the remaining twenty-six acres he allocated for corn.\n\nHow many acres of land did the farmer have?\n\nAnd Alice was able to answer this question correctly. Can you? (No need to use algebra at all!)",
        "answer": "120",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 36540,
        "difficulty": null
      },
      {
        "id": "9a03516281fe2ff8fb3d748b388a35abd0b2a4808dae1e1c02ec97869864167c",
        "domain": "numina",
        "question": "9.5. The heights $A D$ and $B E$ of an acute-angled triangle $A B C$ intersect at point $H$. The circumcircle of triangle $A B H$ intersects sides $A C$ and $B C$ at points $F$ and $G$ respectively. Find $F G$, if $D E=$ $=5$ cm.",
        "answer": "10",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 13131,
        "difficulty": null
      },
      {
        "id": "2913106569c40e75fc7d459e661d31832ddb239e1d24e0c28ac08c0d9de6e7db",
        "domain": "numina",
        "question": "36. The Village Chief of Sheep Village asked Xiao Yangyang, Fei Yangyang, and Lan Yangyang to monitor the grass supply situation of the A, B, and C grasslands on Green Green Prairie. Xiao Yangyang reported: “The grass on A grassland can feed 18 sheep for 6 days.”\nFei Yangyang reported: “The grass on B grassland can feed 30 sheep for 8 days.”\nLan Yangyang was only concerned with sleeping and did not submit a report.\nThe Village Chief said: “I know that the area of B grassland is twice the area of A grassland, and the area of C grassland is three times the area of A grassland. The grass on all three grasslands grows at a uniform rate.”\nMei Yangyang said: “Based on this information, we can calculate that the grass on C grassland can feed 26 sheep for $\\qquad$ days. $\"$",
        "answer": "27",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 146379,
        "difficulty": null
      },
      {
        "id": "d2c2ca33bcdb74579cabbb1e364e65d93b2b59c4be3efa987e45c11b00ec5206",
        "domain": "numina",
        "question": "$3 \\cdot 1$ If the numbers $p, p+10, p+14$ are all prime, find $p$.\n(Kiev Mathematical Olympiad, 1981)",
        "answer": "3",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 94921,
        "difficulty": null
      },
      {
        "id": "3ea9397a9535c43d26e518c69efc59468ed08a52799f5732ceea5f25757ac1ea",
        "domain": "numina",
        "question": "H4. The largest of four different real numbers is $d$. When the numbers are summed in pairs, the four largest sums are $9,10,12$ and 13 .\nWhat are the possible values of $d$ ?",
        "answer": "7.58",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 155242,
        "difficulty": null
      },
      {
        "id": "1c37d4fdede6d1ab0cea1f77fc2e876878f0e51077956533c0c379719bb72e40",
        "domain": "numina",
        "question": "4.3. In a triangle $A B C$ the following relation holds:\n\n$$\n\\sin ^{23} \\frac{\\alpha}{2} \\cdot \\cos ^{48} \\frac{\\beta}{2}=\\sin ^{23} \\frac{\\beta}{2} \\cdot \\cos ^{48} \\frac{\\alpha}{2}\n$$\n\nwhere $\\alpha$ and $\\beta$ are the corresponding angles at $A$ and $B$. Find the ratio $A C / B C$.\n\n(Cyprus)",
        "answer": "1",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 162420,
        "difficulty": null
      },
      {
        "id": "9b557de02a3ab9534cd5f94b3e7d9dc3418fca73bfd7bff14b1ecade6d2ba789",
        "domain": "numina",
        "question": "1. The \"Pioneer\" spacecraft travels back and forth between \"Earth\" and the \"M78 Nebula\". If the speed from \"Earth\" to \"M78 Nebula\" is 30000 kilometers/hour, and the speed from \"M78 Nebula\" to \"Earth\" is 20000 kilometers/hour, then the average speed of the \"Pioneer\" spacecraft for the round trip is $\\qquad$ kilometers/hour.",
        "answer": "24000",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 130718,
        "difficulty": null
      },
      {
        "id": "dad59baf9d4350fae2e328e89c1e853d7fa2cd7a5698da22f3385de6b9f3ea33",
        "domain": "numina",
        "question": "59. (USS 6) On the circle with center $O$ and radius 1 the point $A_{0}$ is fixed and points $A_{1}, A_{2}, \\ldots, A_{999}, A_{1000}$ are distributed in such a way that $\\angle A_{0} O A_{k}=k$ (in radians). Cut the circle at points $A_{0}, A_{1}, \\ldots, A_{1000}$. How many arcs with different lengths are obtained?  ### 3.10 The Tenth IMO",
        "answer": "3",
        "topic": "Geometry",
        "source": "olympiads_ref",
        "source_index": 196643,
        "difficulty": null
      },
      {
        "id": "5f328423bd91fcbff56d0fee89e5b7b15723c45bb00278b302bca9d0358cc34b",
        "domain": "numina",
        "question": "# Problem 1.\n\nIn-1 Find $f(1)+f(2)+f(3)+\\ldots+f(13)$, if $f(n)=4 n^{3}-6 n^{2}+4 n+13$.",
        "answer": "28743",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 6298,
        "difficulty": null
      },
      {
        "id": "a3f23bd61639b3f8d8c60dcd3ae5a523d409424369727ac686135c48971e0f6a",
        "domain": "numina",
        "question": "4. Let the set $A=\\left\\{\\frac{1}{2}, \\frac{1}{7}, \\frac{1}{11}, \\frac{1}{13}, \\frac{1}{15}, \\frac{1}{32}\\right\\}$ have non-empty subsets $A_{1}, A_{2}, \\cdots, A_{63}$, and let the product of all elements in the set $A_{i}(i=1,2$, $\\cdots, 63$) be $p_{i}$ (the product of a single-element set is the element itself). Then\n$$\np_{1}+p_{2}+\\cdots+p_{63}=\n$$\n$\\qquad$",
        "answer": "\\frac{79}{65}",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 117782,
        "difficulty": null
      },
      {
        "id": "27ffd3602cb4e5027797d36e222712506cdc1a5b0d7dd9cd02ee5f51a030198d",
        "domain": "numina",
        "question": "3. (7p) Consider the sequence of real numbers $\\left(x_{n}\\right)_{n \\circledast 1}$ with $x_{1}=1, x_{2}=2, x_{3}=3$ and $x_{n+3}=x_{n}, \\forall n \\in \\mathbb{N}^{*}$. Calculate $\\lim _{n \\rightarrow \\infty} \\frac{x_{1}+x_{2}+\\ldots+x_{n}}{n}$.",
        "answer": "2",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 178343,
        "difficulty": null
      },
      {
        "id": "f2e21d6cdfe4dadbb9d0b3b93ae4b56e4c335da1844822b7bd42e3f8e3e04237",
        "domain": "numina",
        "question": "40. In a football tournament, each of the participating teams played against each other once. The teams scored $16,14,10,10,8,6,5,3$ points. How many teams participated in the tournament and how many points did the teams that played and finished in the top 4 positions lose? (A team gets 2 points for a win and 1 point for a draw.)",
        "answer": "9",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 39349,
        "difficulty": null
      },
      {
        "id": "4671711cefda66420dbf50fccc3c73d0e53cd6aba18156d72b0add115983cea0",
        "domain": "numina",
        "question": "2. Three towns $X, Y$ and $Z$ lie on a plane with coordinates $(0,0),(200,0)$ and $(0,300)$ respectively. There are 100,200 and 300 students in towns $X, Y$ and $Z$ respectively. A school is to be built on a grid point $(x, y)$, where $x$ and $y$ are both integers, such that the overall distance travelled by all the students is minimized. Find the value of $x+y$.",
        "answer": "300",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 160736,
        "difficulty": null
      },
      {
        "id": "e1e66eb1afb5fc181adc73e60c0afc238b1526c40a2ad94c6dd954c6f424252d",
        "domain": "numina",
        "question": "2. The sum of the squares of all real roots of the equation $x^{2}-31 x+220=2^{x}\\left(31-2 x-2^{x}\\right)$ is $\\qquad$",
        "answer": "25",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 118574,
        "difficulty": null
      },
      {
        "id": "a4b109364256517219c00a91e8ae951629aeb4837bd0614e6e239d7f796810d3",
        "domain": "numina",
        "question": "11. From the \"Greek Anthology\". Four springs rise from the ground. The first fills the basin in one day, the second in 2 days, the third in 3 days, and the fourth in 4 days. How long will it take for all four springs to fill the basin?",
        "answer": "\\frac{12}{25}",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 35561,
        "difficulty": null
      },
      {
        "id": "67e76282384d104f0fe0067b53549eeed08b0e2962d0c159531b56afd8c57760",
        "domain": "numina",
        "question": "9.5. In each cell of a 10 by 10 table, a minus sign is written. In one operation, it is allowed to simultaneously change the signs to their opposites in all cells of some column and some row (plus to minus and vice versa). What is the minimum number of operations required to make all the signs in the table pluses?",
        "answer": "100",
        "topic": "Logic and Puzzles",
        "source": "olympiads",
        "source_index": 9960,
        "difficulty": null
      },
      {
        "id": "c2383924248814126433e8c24ccb7d9c23503bc563a88b50278ead712fbb5ac7",
        "domain": "numina",
        "question": "Problem 11.6. Given the set of numbers $\\{-1,-2,-3, \\ldots,-26\\}$. On the board, all possible subsets of this set were written, each containing at least 2 numbers. For each written subset, the product of all numbers belonging to that subset was calculated. What is the sum of all these products?",
        "answer": "350",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 17157,
        "difficulty": null
      },
      {
        "id": "1e826ee0f38d0620aa36cf9b354c08bdd6b8a3b7316ee9b0bd8813d8a6be956b",
        "domain": "numina",
        "question": "7. (10 points) $\\left[x-\\frac{1}{2}\\right]=3 x-5$, here $[x]$ represents the greatest integer not exceeding $x$, then $x=$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 93153,
        "difficulty": null
      },
      {
        "id": "afac848137988a407ee3bcff2f082f6f3834b0ddbf7387b570cd85e29240cebf",
        "domain": "numina",
        "question": "7.111. $\\frac{1}{3} \\lg \\left(271+3^{2 \\sqrt{x}}\\right)+\\lg 10=2$.",
        "answer": "9",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 50873,
        "difficulty": null
      },
      {
        "id": "eabd90e2d640decb9424e839fefa57c462dacf27ee849fad262b0f2500c0b12e",
        "domain": "numina",
        "question": "## Condition of the problem\n\nTo derive the equation of the normal to the given curve at the point with abscissa $x_{0}$.\n\n$y=\\frac{4 x-x^{2}}{4}, x_{0}=2$",
        "answer": "2",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 47019,
        "difficulty": null
      },
      {
        "id": "d15fe691974a858e23685a475672313a31f14bbda3dcd3e48edc0d48e62ef431",
        "domain": "numina",
        "question": "1. (3 points) $2000-180+220-180+220-180+220-180+220-180+220=$",
        "answer": "2200",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 112184,
        "difficulty": null
      },
      {
        "id": "86f908ea054b09be612ec8f13149014c88d234245a9bf95f98d2e574532dab8f",
        "domain": "numina",
        "question": "25th CanMO 1993 Problem 4 Several schools took part in a tournament. Each player played one match against each player from a different school and did not play anyone from the same school. The total number of boys taking part differed from the total number of girls by 1. The total number of matches with both players of the same sex differed by at most one from the total number of matches with players of opposite sex. What is the largest number of schools that could have sent an odd number of players to the tournament? Solution",
        "answer": "3",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 149159,
        "difficulty": null
      },
      {
        "id": "6a5023abca9968aac932e50477a0da1f4858327f2952f1b023feb88a983a5727",
        "domain": "numina",
        "question": "7. (10 points) There are 10 cards each of the numbers “3”, “4”, and “5”. From these, any 8 cards are selected so that their sum is 27. Then, the maximum number of cards that can be “3” is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
        "answer": "6",
        "topic": "Calculus",
        "source": "olympiads",
        "source_index": 144806,
        "difficulty": null
      },
      {
        "id": "b5f8e851063cdfc6dd639a95e76fb864d7323592fc38c70ef6fe0bd2e6ca6071",
        "domain": "numina",
        "question": "Let $A B C D$ and $E F G H$ be squares with side lengths 33 and 12, respectively, with $E F$ lying on side $D C$ (as shown in the figure below). Let $X$ be the intersection point of segments $H B$ and $D C$. Suppose that $\\overline{D E}=18$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_463eeeabc25d438954b1g-17.jpg?height=754&width=542&top_left_y=1209&top_left_x=839)\n\na) Calculate the length of the segment $\\overline{E X}$.\n\nb) Prove that the points $A, X$, and $G$ are collinear.",
        "answer": "4",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 73991,
        "difficulty": null
      },
      {
        "id": "283870998ce138dae7aaa4fa9b8cfde424cbfc44936af485d8d27cc572bf864e",
        "domain": "numina",
        "question": "4. For which bases $b>6$ does the representation 5654 stand for a power of a prime number?",
        "answer": "7",
        "topic": "Number Theory",
        "source": "olympiads",
        "source_index": 181656,
        "difficulty": null
      },
      {
        "id": "1601d27732464703bce6b5862442e45945a65f5f4ff954b9bed1677aa7fad047",
        "domain": "numina",
        "question": "Exercise 5. Martin is trying to fill each cell of a rectangular grid with 8 rows and $\\mathrm{n}$ columns with one of the four letters $\\mathrm{P}, \\mathrm{O}, \\mathrm{F}$, and $\\mathrm{M}$ such that for any pair of distinct rows, there is at most one column where the intersections of the two rows are cells with the same letter. What is the largest integer $n$ for which this is possible?",
        "answer": "7",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 77538,
        "difficulty": null
      },
      {
        "id": "a4d033e2b55f7caeefa58b9f82bafd9bf6fc757d5c44cf0bca323c70eb11c166",
        "domain": "numina",
        "question": "3. The isosceles trapezoid $A B C D$ has parallel sides $A B$ and $C D$. We know that $\\overline{A B}=6, \\overline{A D}=5$ and $\\angle D A B=60^{\\circ}$. A light ray is launched from $A$ that bounces off $C B$ at point $E$ and intersects $A D$ at point $F$. If $A F$ $=3$, calculate the area of triangle $A F E$.",
        "answer": "\\frac{3\\sqrt{3}}{2}",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 86230,
        "difficulty": null
      },
      {
        "id": "94f8b14d15dbf0f552ff9fa543041b5d63456ce2e75e370feba1d9cc10730a62",
        "domain": "numina",
        "question": "Kozhevnikov P.A\n\nOn each of two parallel lines $a$ and $b$, 50 points were marked.\n\nWhat is the maximum possible number of acute-angled triangles with vertices at these points?",
        "answer": "41650",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 63862,
        "difficulty": null
      },
      {
        "id": "5986c43c51f6d78144b7c0c3fb814113db88c673e4fe37350ec1b056acfdf3cf",
        "domain": "numina",
        "question": "6. Given a convex quadrilateral ABCD. Point $M$ is the midpoint of side BC, and point $N$ is the midpoint of side CD. Segments AM, AN, and $MN$ divide the quadrilateral into four triangles, the areas of which, written in some order, are consecutive natural numbers. What is the maximum possible area of triangle $\\mathrm{ABD}$?",
        "answer": "6",
        "topic": "Geometry",
        "source": "olympiads",
        "source_index": 16391,
        "difficulty": null
      },
      {
        "id": "fa675e687c692ec9df4b7d8454eb9127ee3c5d2bf5e0399e409dd32ebb837dbe",
        "domain": "numina",
        "question": "Problem 7. For what values of the parameter $a$ does the equation $x^{3}-15 x^{2}+a x-64=0$ have three distinct real roots that form a geometric progression?",
        "answer": "60",
        "topic": "Algebra",
        "source": "olympiads",
        "source_index": 9326,
        "difficulty": null
      },
      {
        "id": "eb7a117889afb6af644a2a68a95fd158ec3d98b79807087166acd11abdbafa02",
        "domain": "numina",
        "question": "Example 2 In a shooting competition, there are 8 clay targets hanging as shown in Figure 18-1. A marksman shoots the targets according to the following rules:\n(1) First, choose a column from which a target will be shot down;\n(2) Then, in the selected column, shoot down the lowest remaining target.\nHow many different sequences are there to shoot down these 8 targets?",
        "answer": "560",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 108206,
        "difficulty": null
      },
      {
        "id": "e485b5ff7ad9c15fef1a03f27c32deffa2a05c11b207c201a2ea6f51e9da488b",
        "domain": "numina",
        "question": "1. A true-false test has ten questions. If you answer five questions \"true\" and five \"false,\" your score is guaranteed to be at least four. How many answer keys are there for which this is true?",
        "answer": "22",
        "topic": "Combinatorics",
        "source": "olympiads",
        "source_index": 152306,
        "difficulty": null
      }
    ],
    "omni_test": [
      {
        "id": "faf6733e77d99b75eb0fc0a58558b4d4ece3387de9b9b5395cb3b0b9f5085b4f",
        "domain": "omni",
        "question": "The graph of $x^{4}=x^{2} y^{2}$ is a union of $n$ different lines. What is the value of $n$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 467,
        "difficulty": 3.5
      },
      {
        "id": "e7618b32943bc2dbb7c8810a7fbd55d42d5201980e187574d55687edb4a9f9ea",
        "domain": "omni",
        "question": "In how many ways can the positive integers from 1 to 100 be arranged in a circle such that the sum of every two integers placed opposite each other is the same? (Arrangements that are rotations of each other count as the same.)",
        "answer": "49!\\cdot 2^{49}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1678,
        "difficulty": 4.5
      },
      {
        "id": "b480c16042172a37cf9f9360a59c450893788e89247a2f9fbc16d0637c9aa735",
        "domain": "omni",
        "question": "Let $A=H_{1}, B=H_{6}+1$. A real number $x$ is chosen randomly and uniformly in the interval $[A, B]$. Find the probability that $x^{2}>x^{3}>x$.",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 971,
        "difficulty": 3.75
      },
      {
        "id": "69c4adf87630ccfa898ac1bb839fee660e2e6fce0ab96e457db005cb4ad8eaee",
        "domain": "omni",
        "question": "Suppose Harvard Yard is a $17 \\times 17$ square. There are 14 dorms located on the perimeter of the Yard. If $s$ is the minimum distance between two dorms, the maximum possible value of $s$ can be expressed as $a-\\sqrt{b}$ where $a, b$ are positive integers. Compute $100a+b$.",
        "answer": "602",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "HMMT_11",
        "source_index": 2498,
        "difficulty": 5.25
      },
      {
        "id": "4dd73ec1ef61502bf0f505c1c41540d89b5795eba555aeafd1df78df0f5520ab",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that, if there are initially $n+1$ townspeople and $n$ goons, then the probability the townspeople win is less than $1\\%$.",
        "answer": "6",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2061,
        "difficulty": 4.0
      },
      {
        "id": "cba03a651d4a1bde9f05d4add2a1346426b58edf8c9fa53458b7d8f3511eda3f",
        "domain": "omni",
        "question": "How many integers $n>1$ are there such that $n$ divides $x^{13}-x$ for every positive integer $x$?",
        "answer": "31",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "rioplatense_mathematical_olympiad_level",
        "source_index": 3811,
        "difficulty": 7.0
      },
      {
        "id": "7e709853f9cd697257c56184c4e9a65af22dae6005abeb3c387aaebd9ea4cc28",
        "domain": "omni",
        "question": "A domino is a 1-by-2 or 2-by-1 rectangle. A domino tiling of a region of the plane is a way of covering it (and only it) completely by nonoverlapping dominoes. For instance, there is one domino tiling of a 2-by-1 rectangle and there are 2 tilings of a 2-by-2 rectangle (one consisting of two horizontal dominoes and one consisting of two vertical dominoes). How many domino tilings are there of a 2-by-10 rectangle?",
        "answer": "89",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 750,
        "difficulty": 3.5
      },
      {
        "id": "ba2bfed9480de13f7e1a8344370cd638987f67ef7672b67aeb75506f39cc47e6",
        "domain": "omni",
        "question": "Consider an infinite grid of unit squares. An $n$-omino is a subset of $n$ squares that is connected. Below are depicted examples of 8 -ominoes. Two $n$-ominoes are considered equivalent if one can be obtained from the other by translations and rotations. What is the number of distinct 15 -ominoes? Your score will be equal to $25-13|\\ln (A)-\\ln (C)|$.",
        "answer": "3426576",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 2456,
        "difficulty": 5.25
      },
      {
        "id": "d69cb7ac62b9580bcf35159e2141d78bfa3d4abfbbef57b9f8851961fd7f40d4",
        "domain": "omni",
        "question": "What is the quantity equivalent to '2% of 1'?",
        "answer": "\\frac{2}{100}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 3118,
        "difficulty": 1.5
      },
      {
        "id": "71b02860335eb3f3f215f289fd497125fc1ad5082ffce2123108a8d204c22bf2",
        "domain": "omni",
        "question": "Suppose that there are real numbers $a, b, c \\geq 1$ and that there are positive reals $x, y, z$ such that $$\\begin{aligned} a^{x}+b^{y}+c^{z} & =4 \\\\ x a^{x}+y b^{y}+z c^{z} & =6 \\\\ x^{2} a^{x}+y^{2} b^{y}+z^{2} c^{z} & =9 \\end{aligned}$$ What is the maximum possible value of $c$ ?",
        "answer": "\\sqrt[3]{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 2657,
        "difficulty": 5.0
      },
      {
        "id": "f41faf32a1012e90aeade402ee2303e31c181378f6e56d0fd7ee32b82f373ff0",
        "domain": "omni",
        "question": "For all $n \\geq 1$, let \\[ a_n = \\sum_{k=1}^{n-1} \\frac{\\sin \\left( \\frac{(2k-1)\\pi}{2n} \\right)}{\\cos^2 \\left( \\frac{(k-1)\\pi}{2n} \\right) \\cos^2 \\left( \\frac{k\\pi}{2n} \\right)}. \\] Determine \\[ \\lim_{n \\to \\infty} \\frac{a_n}{n^3}. \\]",
        "answer": "\\frac{8}{\\pi^3}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Other",
        "source": "putnam",
        "source_index": 3177,
        "difficulty": 8.0
      },
      {
        "id": "e31eb63833f591e17ac75e9c3e6d1fa2d275930eddcb4fa9062c114e32bcaee2",
        "domain": "omni",
        "question": "Ken is the best sugar cube retailer in the nation. Trevor, who loves sugar, is coming over to make an order. Ken knows Trevor cannot afford more than 127 sugar cubes, but might ask for any number of cubes less than or equal to that. Ken prepares seven cups of cubes, with which he can satisfy any order Trevor might make. How many cubes are in the cup with the most sugar?",
        "answer": "64",
        "topic": "Mathematics -> Applied Mathematics -> Word Problems -> Other",
        "source": "HMMT_2",
        "source_index": 341,
        "difficulty": 3.5
      },
      {
        "id": "cb4a35abad5e6f6553f39fa6079996e73f5fdc3c6928f87914785fdfcd6615da",
        "domain": "omni",
        "question": "Emilia writes down the numbers $5, x$, and 9. Valentin calculates the mean (average) of each pair of these numbers and obtains 7, 10, and 12. What is the value of $x$?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2731,
        "difficulty": 2.5
      },
      {
        "id": "ff1cd63e7b40e5f9641bb0cee80567d7f62c51668afe07c32ac34cbfc90f6001",
        "domain": "omni",
        "question": "Define a sequence of polynomials as follows: let $a_{1}=3 x^{2}-x$, let $a_{2}=3 x^{2}-7 x+3$, and for $n \\geq 1$, let $a_{n+2}=\\frac{5}{2} a_{n+1}-a_{n}$. As $n$ tends to infinity, what is the limit of the sum of the roots of $a_{n}$ ?",
        "answer": "\\frac{13}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 2351,
        "difficulty": 5.0
      },
      {
        "id": "bdb011da7e474caf2617d9b4a2dbd7a2aacfc1cafc63c8a3709523d4b1fb37a3",
        "domain": "omni",
        "question": "Three integers from the list $1,2,4,8,16,20$ have a product of 80. What is the sum of these three integers?",
        "answer": "25",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3018,
        "difficulty": 1.5
      },
      {
        "id": "43b7057960bce346884c68ac8e59d9176dc008096a5da721c5648d56a399fae7",
        "domain": "omni",
        "question": "Let $f(n)$ be the number of distinct digits of $n$ when written in base 10. Compute the sum of $f(n)$ as $n$ ranges over all positive 2019-digit integers.",
        "answer": "9\\left(10^{2019}-9^{2019}\\right)",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2071,
        "difficulty": 5.25
      },
      {
        "id": "07ebfbb358bc26c4d867a3d3faf25627c8fbbf02d3c71477fbd680bbf52e55b7",
        "domain": "omni",
        "question": "A fat coin is one which, when tossed, has a $2 / 5$ probability of being heads, $2 / 5$ of being tails, and $1 / 5$ of landing on its edge. Mr. Fat starts at 0 on the real line. Every minute, he tosses a fat coin. If it's heads, he moves left, decreasing his coordinate by 1; if it's tails, he moves right, increasing his coordinate by 1. If the coin lands on its edge, he moves back to 0. If Mr. Fat does this ad infinitum, what fraction of his time will he spend at 0?",
        "answer": "\\[\n\\frac{1}{3}\n\\]",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 3301,
        "difficulty": 5.0
      },
      {
        "id": "950e1f1bfa489a2c9970c213f1c2ab520b9d92292b0acbbd238aae800ac4a29f",
        "domain": "omni",
        "question": "If \\( 8 + 6 = n + 8 \\), what is the value of \\( n \\)?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2879,
        "difficulty": 1.0
      },
      {
        "id": "d6a7f54871c048530a786ba61210752ca5d2f0c93cd4519ac98ab6f4adb4157b",
        "domain": "omni",
        "question": "Let \\(ABCD\\) be a square of side length 2. Let points \\(X, Y\\), and \\(Z\\) be constructed inside \\(ABCD\\) such that \\(ABX, BCY\\), and \\(CDZ\\) are equilateral triangles. Let point \\(W\\) be outside \\(ABCD\\) such that triangle \\(DAW\\) is equilateral. Let the area of quadrilateral \\(WXYZ\\) be \\(a+\\sqrt{b}\\), where \\(a\\) and \\(b\\) are integers. Find \\(a+b\\).",
        "answer": "10",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2358,
        "difficulty": 4.5
      },
      {
        "id": "fa92848d069bfee68e3e45af549f04dad1190f5b150690125ca1614e25f693cb",
        "domain": "omni",
        "question": "Given that $a, b, c$ are integers with $a b c=60$, and that complex number $\\omega \\neq 1$ satisfies $\\omega^{3}=1$, find the minimum possible value of $\\left|a+b \\omega+c \\omega^{2}\\right|$.",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2167,
        "difficulty": 4.5
      },
      {
        "id": "c6224ed9f429d69bc76a90192fbdd0392f23f786e9fd604616c813966c7c2979",
        "domain": "omni",
        "question": "Fifteen freshmen are sitting in a circle around a table, but the course assistant (who remains standing) has made only six copies of today's handout. No freshman should get more than one handout, and any freshman who does not get one should be able to read a neighbor's. If the freshmen are distinguishable but the handouts are not, how many ways are there to distribute the six handouts subject to the above conditions?",
        "answer": "125",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1128,
        "difficulty": 4.0
      },
      {
        "id": "a935140cd17472907244e4d8a4cb9bfa92cd6dfa730a8913452c0113a318c71b",
        "domain": "omni",
        "question": "Let $\\triangle A B C$ be a triangle inscribed in a unit circle with center $O$. Let $I$ be the incenter of $\\triangle A B C$, and let $D$ be the intersection of $B C$ and the angle bisector of $\\angle B A C$. Suppose that the circumcircle of $\\triangle A D O$ intersects $B C$ again at a point $E$ such that $E$ lies on $I O$. If $\\cos A=\\frac{12}{13}$, find the area of $\\triangle A B C$.",
        "answer": "\\frac{15}{169}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 563,
        "difficulty": 5.25
      },
      {
        "id": "01b270b1dfa50bd5c0f08c1e352e625eb191ce498e76b5b41ad05008cbd78cce",
        "domain": "omni",
        "question": "$A B C$ is a triangle with points $E, F$ on sides $A C, A B$, respectively. Suppose that $B E, C F$ intersect at $X$. It is given that $A F / F B=(A E / E C)^{2}$ and that $X$ is the midpoint of $B E$. Find the ratio $C X / X F$.",
        "answer": "\\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 807,
        "difficulty": 4.5
      },
      {
        "id": "2ff51e8673fc7f75c25f53c10f40a97e805ed0dead9b5ba8ebb73bc1d7f7324a",
        "domain": "omni",
        "question": "Let $ABCDEF$ be a regular hexagon. Let $P$ be the circle inscribed in $\\triangle BDF$. Find the ratio of the area of circle $P$ to the area of rectangle $ABDE$.",
        "answer": "\\frac{\\pi \\sqrt{3}}{12}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 883,
        "difficulty": 4.0
      },
      {
        "id": "6d3a3568d0d4fd11be4d8635708cfdaaffe813a6d6bd63908f0fb0794aed9aee",
        "domain": "omni",
        "question": "The side lengths of a triangle are distinct positive integers. One of the side lengths is a multiple of 42, and another is a multiple of 72. What is the minimum possible length of the third side?",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2087,
        "difficulty": 4.0
      },
      {
        "id": "d4474708a62ab076f836ec1d2865591a5fd0207ee8c8a1d17819728c2791f1de",
        "domain": "omni",
        "question": "A regular octahedron has a side length of 1. What is the distance between two opposite faces?",
        "answer": "\\sqrt{6} / 3",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1020,
        "difficulty": 3.5
      },
      {
        "id": "87cfeeaf97f53a0ef5932c5fbd00489b45161d13d2326230bff632126b24151f",
        "domain": "omni",
        "question": "Let $f(x)=x^{2}+a x+b$ and $g(x)=x^{2}+c x+d$ be two distinct real polynomials such that the $x$-coordinate of the vertex of $f$ is a root of $g$, the $x$-coordinate of the vertex of $g$ is a root of $f$ and both $f$ and $g$ have the same minimum value. If the graphs of the two polynomials intersect at the point (2012, - 2012), what is the value of $a+c$ ?",
        "answer": "-8048",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 748,
        "difficulty": 5.0
      },
      {
        "id": "9c30b959e4abed743ac86c8063a8f6f7831a8b006c79ef124932c2a1c7973990",
        "domain": "omni",
        "question": "There are 5 students on a team for a math competition. The math competition has 5 subject tests. Each student on the team must choose 2 distinct tests, and each test must be taken by exactly two people. In how many ways can this be done?",
        "answer": "2040",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1052,
        "difficulty": 4.0
      },
      {
        "id": "ea297c58c2ca3709403ee3e6037337fba45ceaf68072c26bb43fa45d0d56ed39",
        "domain": "omni",
        "question": "There are $F$ fractions $\\frac{m}{n}$ with the properties: $m$ and $n$ are positive integers with $m<n$, $\\frac{m}{n}$ is in lowest terms, $n$ is not divisible by the square of any integer larger than 1, and the shortest sequence of consecutive digits that repeats consecutively and indefinitely in the decimal equivalent of $\\frac{m}{n}$ has length 6. We define $G=F+p$, where the integer $F$ has $p$ digits. What is the sum of the squares of the digits of $G$?",
        "answer": "181",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2724,
        "difficulty": 2.5
      },
      {
        "id": "2d0555d201f14c1f434582293df4f31929ea9d2e88565b617762c8ddc026758f",
        "domain": "omni",
        "question": "A counter begins at 0 . Then, every second, the counter either increases by 1 or resets back to 0 with equal probability. The expected value of the counter after ten seconds can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "103324",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2648,
        "difficulty": 5.0
      },
      {
        "id": "516c56434db67ab60b12eec2318efc1216fbd1f445cbdaa9608cce4919c867dd",
        "domain": "omni",
        "question": "Let $f(n)$ and $g(n)$ be polynomials of degree 2014 such that $f(n)+(-1)^{n} g(n)=2^{n}$ for $n=1,2, \\ldots, 4030$. Find the coefficient of $x^{2014}$ in $g(x)$.",
        "answer": "\\frac{3^{2014}}{2^{2014} \\cdot 2014!}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 900,
        "difficulty": 5.0
      },
      {
        "id": "d1435ad38ea7fc68a05653891981a8bb0d9fbb59f47f3a232dba610bd9d4fb15",
        "domain": "omni",
        "question": "Evaluate $1201201_{-4}$.",
        "answer": "2017",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2133,
        "difficulty": 4.0
      },
      {
        "id": "4ae3e3ae1f183bb5f3e74a567272c0b5bbe3567d21247bab14cfc104aa627503",
        "domain": "omni",
        "question": "The real numbers $x, y$ and $z$ satisfy the three equations $x+y=7$, $xz=-180$, and $(x+y+z)^{2}=4$. If $S$ is the sum of the two possible values of $y$, what is $-S$?",
        "answer": "42",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 3111,
        "difficulty": 2.5
      },
      {
        "id": "fa7ad61378930a20adfde8e2b12b060311d5b821d40d8e16232f87dc37f100cf",
        "domain": "omni",
        "question": "Calculate the value of the expression $\\left(2 \\times \\frac{1}{3}\\right) \\times \\left(3 \\times \\frac{1}{2}\\right)$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2696,
        "difficulty": 1.0
      },
      {
        "id": "2e66ec0d981176feac80b298d34d0aede11512042fb91417cb90ccf2bb5dc9eb",
        "domain": "omni",
        "question": "A collection of $n$ squares on the plane is called tri-connected if the following criteria are satisfied: (i) All the squares are congruent. (ii) If two squares have a point $P$ in common, then $P$ is a vertex of each of the squares. (iii) Each square touches exactly three other squares. How many positive integers $n$ are there with $2018 \\leq n \\leq 3018$, such that there exists a collection of $n$ squares that is tri-connected?",
        "answer": "501",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "apmoapmo_sol",
        "source_index": 1602,
        "difficulty": 7.0
      },
      {
        "id": "ec04b2d2b6590160525ce323701111aad83f570634c5184bd55a0e94097d3e30",
        "domain": "omni",
        "question": "How many of the 20 perfect squares $1^{2}, 2^{2}, 3^{2}, \\ldots, 19^{2}, 20^{2}$ are divisible by 9?",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 3012,
        "difficulty": 2.0
      },
      {
        "id": "9e2e1eca52fdc8eecf63593419f64b70cc591ad76ee9aefd65993b7ee4fb5311",
        "domain": "omni",
        "question": "A class of 10 students took a math test. Each problem was solved by exactly 7 of the students. If the first nine students each solved 4 problems, how many problems did the tenth student solve?",
        "answer": "6",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1153,
        "difficulty": 4.0
      },
      {
        "id": "863cabd101dbe544ae9e17438ccf24742ef96d8c364bad48c61a351943a60a76",
        "domain": "omni",
        "question": "Let $A_{10}$ denote the answer to problem 10. Two circles lie in the plane; denote the lengths of the internal and external tangents between these two circles by $x$ and $y$, respectively. Given that the product of the radii of these two circles is $15 / 2$, and that the distance between their centers is $A_{10}$, determine $y^{2}-x^{2}$.",
        "answer": "30",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 2091,
        "difficulty": 4.5
      },
      {
        "id": "afec557d88a352c8b258146ce4efd5a5f12aff59371e7af6a066d2f59de28dc0",
        "domain": "omni",
        "question": "Given a point $p$ and a line segment $l$, let $d(p, l)$ be the distance between them. Let $A, B$, and $C$ be points in the plane such that $A B=6, B C=8, A C=10$. What is the area of the region in the $(x, y)$-plane formed by the ordered pairs $(x, y)$ such that there exists a point $P$ inside triangle $A B C$ with $d(P, A B)+x=d(P, B C)+y=d(P, A C)$?",
        "answer": "\\frac{288}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 596,
        "difficulty": 5.0
      },
      {
        "id": "356811e776c461ba863ed2f7a4ef44ba4f2104971c422e3ef307b292474efe7b",
        "domain": "omni",
        "question": "For the sequence of numbers $n_{1}, n_{2}, n_{3}, \\ldots$, the relation $n_{i}=2 n_{i-1}+a$ holds for all $i>1$. If $n_{2}=5$ and $n_{8}=257$, what is $n_{5}$ ?",
        "answer": "33",
        "topic": "Mathematics -> Algebra -> Sequences and Series -> Other",
        "source": "HMMT_2",
        "source_index": 521,
        "difficulty": 4.0
      },
      {
        "id": "68ee993b45f2a10eaf880b8a74b212db0cbac2790aba5dfcda3afc75502965b5",
        "domain": "omni",
        "question": "When $(3 + 2x + x^{2})(1 + mx + m^{2}x^{2})$ is expanded and fully simplified, the coefficient of $x^{2}$ is equal to 1. What is the sum of all possible values of $m$?",
        "answer": "-\\frac{2}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "fermat",
        "source_index": 3031,
        "difficulty": 2.5
      },
      {
        "id": "7d9c9be86160ebe258024fc39e9c94d438a84fc817dd5f813cddb4cc39e80ded",
        "domain": "omni",
        "question": "Let $ABCD$ be a regular tetrahedron, and let $O$ be the centroid of triangle $BCD$. Consider the point $P$ on $AO$ such that $P$ minimizes $PA+2(PB+PC+PD)$. Find $\\sin \\angle PBO$.",
        "answer": "\\frac{1}{6}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 583,
        "difficulty": 5.0
      },
      {
        "id": "410a500aafa120190d354495009d56a0a8cdd907c6e87f6e842b6289597bdc3a",
        "domain": "omni",
        "question": "Reimu and Sanae play a game using 4 fair coins. Initially both sides of each coin are white. Starting with Reimu, they take turns to color one of the white sides either red or green. After all sides are colored, the 4 coins are tossed. If there are more red sides showing up, then Reimu wins, and if there are more green sides showing up, then Sanae wins. However, if there is an equal number of red sides and green sides, then neither of them wins. Given that both of them play optimally to maximize the probability of winning, what is the probability that Reimu wins?",
        "answer": "\\frac{5}{16}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1180,
        "difficulty": 4.0
      },
      {
        "id": "43370c3c86cba1eada143b882f64a744a3e58b3c9bcee4dd9672905b8b9a56ee",
        "domain": "omni",
        "question": "When a single number is added to each member of the sequence 20, 50, 100, the sequence becomes expressible as $x, a x, a^{2} x$. Find $a$.",
        "answer": "\\frac{5}{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 584,
        "difficulty": 3.5
      },
      {
        "id": "1ce57dd133536a46025c7bf81c158553e4b410363350320ce8041c0670f69501",
        "domain": "omni",
        "question": "The integers $1,2, \\ldots, 64$ are written in the squares of a $8 \\times 8$ chess board, such that for each $1 \\leq i<64$, the numbers $i$ and $i+1$ are in squares that share an edge. What is the largest possible sum that can appear along one of the diagonals?",
        "answer": "432",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1077,
        "difficulty": 5.5
      },
      {
        "id": "37555ac5a6c63108a9f0c47ddce11af2a4bdf1933852617e5f5fe990b7ee159f",
        "domain": "omni",
        "question": "Let a sequence $\\left\\{a_{n}\\right\\}_{n=0}^{\\infty}$ be defined by $a_{0}=\\sqrt{2}, a_{1}=2$, and $a_{n+1}=a_{n} a_{n-1}^{2}$ for $n \\geq 1$. The sequence of remainders when $a_{0}, a_{1}, a_{2}, \\cdots$ are divided by 2014 is eventually periodic with some minimal period $p$ (meaning that $a_{m}=a_{m+p}$ for all sufficiently large integers $m$, and $p$ is the smallest such positive integer). Find $p$.",
        "answer": "12",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 1729,
        "difficulty": 6.0
      },
      {
        "id": "6d1a367c0757acd4c8f52da6987cad7f487dca8f4ba74e5986d46d3ff5da4719",
        "domain": "omni",
        "question": "Fran writes the numbers \\(1,2,3, \\ldots, 20\\) on a chalkboard. Then she erases all the numbers by making a series of moves; in each move, she chooses a number \\(n\\) uniformly at random from the set of all numbers still on the chalkboard, and then erases all of the divisors of \\(n\\) that are still on the chalkboard (including \\(n\\) itself). What is the expected number of moves that Fran must make to erase all the numbers?",
        "answer": "\\frac{131}{10}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1619,
        "difficulty": 5.0
      },
      {
        "id": "61c6a0b5f434c7209221f166802eb691ee7eb04b5014c75dbc5f809d397d4c59",
        "domain": "omni",
        "question": "Find the number of subsets $S$ of $\\{1,2, \\ldots 63\\}$ the sum of whose elements is 2008.",
        "answer": "66",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 663,
        "difficulty": 5.0
      },
      {
        "id": "a8928df66c2adc1c24800d713f43af20dc03616f71f8c8aee9c5437bb2e2a5b6",
        "domain": "omni",
        "question": "Digits are placed in the two boxes of $2 \\square \\square$, with one digit in each box, to create a three-digit positive integer. In how many ways can this be done so that the three-digit positive integer is larger than 217?",
        "answer": "82",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2769,
        "difficulty": 2.0
      },
      {
        "id": "1f45e4be4de87af3372da3e09519644aa99ece229c9ef50d29e1ab05e39316ec",
        "domain": "omni",
        "question": "For each positive integer $n$, let $k(n)$ be the number of ones in the binary representation of $2023 \\cdot n$. What is the minimum value of $k(n)$?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Binary Representation -> Other; Mathematics -> Number Theory -> Modular Arithmetic -> Other; Mathematics -> Number Theory -> Factorization",
        "source": "putnam",
        "source_index": 3278,
        "difficulty": 7.0
      },
      {
        "id": "18a1f95b87cf9f45496df971ae704be51effdffb6989430e3ee2b1567f9f0931",
        "domain": "omni",
        "question": "Ash and Gary independently come up with their own lineups of 15 fire, grass, and water monsters. Then, the first monster of both lineups will fight, with fire beating grass, grass beating water, and water beating fire. The defeated monster is then substituted with the next one from their team's lineup; if there is a draw, both monsters get defeated. Gary completes his lineup randomly, with each monster being equally likely to be any of the three types. Without seeing Gary's lineup, Ash chooses a lineup that maximizes the probability $p$ that his monsters are the last ones standing. Compute $p$.",
        "answer": "1-\\frac{2^{15}}{3^{15}}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 787,
        "difficulty": 5.25
      },
      {
        "id": "c11a5f88c303ce4b46baa6837c9cb22241b0b06f06b54f0630c44adb7db5facf",
        "domain": "omni",
        "question": "The operation $\\nabla$ is defined by $g \\nabla h=g^{2}-h^{2}$. If $g>0$ and $g \\nabla 6=45$, what is the value of $g$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3053,
        "difficulty": 1.5
      },
      {
        "id": "411214066a3e8510ad9341cc5bcb75721862c07f3255307ef63379e881f16e36",
        "domain": "omni",
        "question": "Find the smallest integer $k \\geq 2$ such that for every partition of the set $\\{2, 3,\\hdots, k\\}$ into two parts, at least one of these parts contains (not necessarily distinct) numbers $a$, $b$ and $c$ with $ab = c$.",
        "answer": "32",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "baltic_way",
        "source_index": 3561,
        "difficulty": 6.0
      },
      {
        "id": "854bd30914d7dd0ebf95d4a382889f6e6651ec95c3ee4775ed4bc90a4ca1ba91",
        "domain": "omni",
        "question": "Harriet ran a 1000 m course in 380 seconds. She ran the first 720 m of the course at a constant speed of 3 m/s. What was her speed for the remaining part of the course?",
        "answer": "2",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3506,
        "difficulty": 2.0
      },
      {
        "id": "52a0703d836b55f703307cbd02b644a30813ad6f62f18888108da9a7a5e100f9",
        "domain": "omni",
        "question": "An acute triangle $ABC$ is inscribed in a circle of radius 1 with centre $O;$ all the angles of $ABC$ are greater than $45^\\circ.$ \n$B_{1}$ is the foot of perpendicular from $B$ to $CO,$ $B_{2}$ is the foot of perpendicular from $B_{1}$ to $AC.$ \nSimilarly, $C_{1}$ is the foot of perpendicular from $C$ to $BO,$ $C_{2}$ is the foot of perpendicular from $C_{1}$ to $AB.$ \nThe lines $B_{1}B_{2}$ and $C_{1}C_{2}$ intersect at $A_{3}.$ The points $B_{3}$ and $C_{3}$ are defined in the same way. \nFind the circumradius of triangle $A_{3}B_{3}C_{3}.$",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "tuymaada_olympiad",
        "source_index": 3906,
        "difficulty": 6.0
      },
      {
        "id": "65d9a1ba653d029bb09eb246ceb67b313988c9499058cfea730d382f45a4a95f",
        "domain": "omni",
        "question": "Each of given $100$ numbers was increased by $1$. Then each number was increased by $1$ once more. Given that the fi\frst time the sum of the squares of the numbers was not changed \ffind how this sum was changed the second time.",
        "answer": "200",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "ToT",
        "source_index": 3997,
        "difficulty": 5.0
      },
      {
        "id": "88040d45473026a79d3f873fc3ed463fc5c28c308a16e36f0739fa77929f1215",
        "domain": "omni",
        "question": "An ordered pair $(a, b)$ of positive integers is called spicy if $\\operatorname{gcd}(a+b, ab+1)=1$. Compute the probability that both $(99, n)$ and $(101, n)$ are spicy when $n$ is chosen from $\\{1,2, \\ldots, 2024\\}$ uniformly at random.",
        "answer": "\\frac{96}{595}",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 741,
        "difficulty": 5.0
      },
      {
        "id": "5eef0874ca60cd6734a75ca31a59e4c5d8a912d3390c3181f80fa5766edca697",
        "domain": "omni",
        "question": "Determine the smallest positive real number $r$ such that there exist differentiable functions $f\\colon \\mathbb{R} \\to \\mathbb{R}$ and $g\\colon \\mathbb{R} \\to \\mathbb{R}$ satisfying \\begin{enumerate} \\item[(a)] $f(0) > 0$, \\item[(b)] $g(0) = 0$, \\item[(c)] $|f'(x)| \\leq |g(x)|$ for all $x$, \\item[(d)] $|g'(x)| \\leq |f(x)|$ for all $x$, and \\item[(e)] $f(r) = 0$. \\end{enumerate}",
        "answer": "\\frac{\\pi}{2}",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives; Mathematics -> Calculus -> Differential Calculus -> Derivatives",
        "source": "putnam",
        "source_index": 3270,
        "difficulty": 7.5
      },
      {
        "id": "293f691fcf32bef4c4a6571297bcf7f27119b21924ef74018510e4bd5e7bba57",
        "domain": "omni",
        "question": "How many six-digit multiples of 27 have only 3, 6, or 9 as their digits?",
        "answer": "51",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2475,
        "difficulty": 5.0
      },
      {
        "id": "8317f1205685f91cde8667a2f75a4cd539526b3832f1d6ce7446b0bbea57a39b",
        "domain": "omni",
        "question": "What number did Janet pick if she added 7 to the number, multiplied the sum by 2, subtracted 4, and the final result was 28?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3483,
        "difficulty": 1.5
      },
      {
        "id": "492f6cb83333b46e4f490fafc167c8d3307d1c7e596db803f77c0a27d03dc1c4",
        "domain": "omni",
        "question": "Alice starts with the number 0. She can apply 100 operations on her number. In each operation, she can either add 1 to her number, or square her number. After applying all operations, her score is the minimum distance from her number to any perfect square. What is the maximum score she can attain?",
        "answer": "94",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1892,
        "difficulty": 4.5
      },
      {
        "id": "e729e14f406b926c956585549c36c037cc3a17165b28be51330bf984a371c329",
        "domain": "omni",
        "question": "Let $\\triangle A B C$ be an acute triangle, with $M$ being the midpoint of $\\overline{B C}$, such that $A M=B C$. Let $D$ and $E$ be the intersection of the internal angle bisectors of $\\angle A M B$ and $\\angle A M C$ with $A B$ and $A C$, respectively. Find the ratio of the area of $\\triangle D M E$ to the area of $\\triangle A B C$.",
        "answer": "\\frac{2}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1888,
        "difficulty": 5.0
      },
      {
        "id": "d056e7059608a050330213045d62e212f99e5c4ee1237246daf57f64199d5452",
        "domain": "omni",
        "question": "Two positive integers \\( x \\) and \\( y \\) have \\( xy=24 \\) and \\( x-y=5 \\). What is the value of \\( x+y \\)?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2798,
        "difficulty": 2.5
      },
      {
        "id": "dbc026dda1666753ccd946dbf6d95629d6369adcb01ca31ab5c7cc46d37cceda",
        "domain": "omni",
        "question": "Let $a, b, c$ be non-negative numbers with $a+b+c = 3$. Prove the inequality\n\\[\\frac{a}{b^2+1}+\\frac{b}{c^2+1}+\\frac{c}{a^2+1} \\geq \\frac 32.\\]",
        "answer": "\\frac{3}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "mediterranean_mathematics_olympiad",
        "source_index": 3719,
        "difficulty": 6.0
      },
      {
        "id": "7c27ca6ee04edfbe0d7a144bd85c8a81e54d1b372a0ee8ca8810dc9e475ccd7b",
        "domain": "omni",
        "question": "How many ways are there to arrange the numbers $1,2,3,4,5,6$ on the vertices of a regular hexagon such that exactly 3 of the numbers are larger than both of their neighbors? Rotations and reflections are considered the same.",
        "answer": "8",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2382,
        "difficulty": 4.0
      },
      {
        "id": "4b655d3cde8021ba3058fb1dfd858543b626a2035ec5bbe8efe0686d1b612075",
        "domain": "omni",
        "question": "Regular tetrahedron $A B C D$ is projected onto a plane sending $A, B, C$, and $D$ to $A^{\\prime}, B^{\\prime}, C^{\\prime}$, and $D^{\\prime}$ respectively. Suppose $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}$ is a convex quadrilateral with $A^{\\prime} B^{\\prime}=A^{\\prime} D^{\\prime}$ and $C^{\\prime} B^{\\prime}=C^{\\prime} D^{\\prime}$, and suppose that the area of $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}=4$. Given these conditions, the set of possible lengths of $A B$ consists of all real numbers in the interval $[a, b)$. Compute $b$.",
        "answer": "2 \\sqrt[4]{6}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1084,
        "difficulty": 4.5
      },
      {
        "id": "5e12eca35e62e5389a33795e6bf4743bf099186b993113413e90212a0a78f96a",
        "domain": "omni",
        "question": "Given that $a, b, c$ are positive integers satisfying $$a+b+c=\\operatorname{gcd}(a, b)+\\operatorname{gcd}(b, c)+\\operatorname{gcd}(c, a)+120$$ determine the maximum possible value of $a$.",
        "answer": "240",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 890,
        "difficulty": 5.25
      },
      {
        "id": "2accf6c81a2a0b03a13751f6b58406bd8a93dd4acfe328219e0efa4888e089e6",
        "domain": "omni",
        "question": "Positive integers $a, b$, and $c$ have the property that $a^{b}, b^{c}$, and $c^{a}$ end in 4, 2, and 9, respectively. Compute the minimum possible value of $a+b+c$.",
        "answer": "17",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 676,
        "difficulty": 5.0
      },
      {
        "id": "eb9f61a76501b31be63d7fe071afb5f4da762baa74f0a77e4b5bb974de24765f",
        "domain": "omni",
        "question": "Let $A B C D$ be an isosceles trapezoid with parallel bases $A B=1$ and $C D=2$ and height 1. Find the area of the region containing all points inside $A B C D$ whose projections onto the four sides of the trapezoid lie on the segments formed by $A B, B C, C D$ and $D A$.",
        "answer": "\\frac{5}{8}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1842,
        "difficulty": 4.0
      },
      {
        "id": "1c523fed2397d7b11c9001fb218a09906baca30f5eea3cfe21bc38a0ae9a7e45",
        "domain": "omni",
        "question": "Let $S$ be the set of $3^{4}$ points in four-dimensional space where each coordinate is in $\\{-1,0,1\\}$. Let $N$ be the number of sequences of points $P_{1}, P_{2}, \\ldots, P_{2020}$ in $S$ such that $P_{i} P_{i+1}=2$ for all $1 \\leq i \\leq 2020$ and $P_{1}=(0,0,0,0)$. (Here $P_{2021}=P_{1}$.) Find the largest integer $n$ such that $2^{n}$ divides $N$.",
        "answer": "4041",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Algebra -> Abstract Algebra -> Other (Recurrence Relations) -> Other; Mathematics -> Algebra -> Other (Number Theory - Divisibility) -> Other",
        "source": "HMMT_2",
        "source_index": 528,
        "difficulty": 5.0
      },
      {
        "id": "c5e2a22aaf2bd1c3f8b10fd916be1bcfe0fdd6520813e03f13ffc1bb9a77fa7e",
        "domain": "omni",
        "question": "For each positive integer $n$, define $S(n)$ to be the smallest positive integer divisible by each of the positive integers $1, 2, 3, \\ldots, n$. How many positive integers $n$ with $1 \\leq n \\leq 100$ have $S(n) = S(n+4)$?",
        "answer": "11",
        "topic": "Mathematics -> Number Theory -> Least Common Multiples (LCM); Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3084,
        "difficulty": 2.5
      },
      {
        "id": "96673addd9b6be1a72710c8497293be0ff867d46546f6d11d6e047a01192ce02",
        "domain": "omni",
        "question": "There are 2018 frogs in a pool and there is 1 frog on the shore. In each time-step thereafter, one random frog moves position. If it was in the pool, it jumps to the shore, and vice versa. Find the expected number of time-steps before all frogs are in the pool for the first time.",
        "answer": "2^{2018}-1",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1579,
        "difficulty": 5.0
      },
      {
        "id": "b81a1c4e473a04c4c4547c196f4a488992896735241d0dbae437e95953f06d7b",
        "domain": "omni",
        "question": "Consider the parabola consisting of the points $(x, y)$ in the real plane satisfying $(y+x)=(y-x)^{2}+3(y-x)+3$. Find the minimum possible value of $y$.",
        "answer": "-\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "HMMT_11",
        "source_index": 2052,
        "difficulty": 4.0
      },
      {
        "id": "7c82091e7734e393a9bd044276510bca0a32906d293dcba3bb675625641fdb0d",
        "domain": "omni",
        "question": "How many ways are there to choose 2010 functions $f_{1}, \\ldots, f_{2010}$ from \\{0,1\\} to \\{0,1\\} such that $f_{2010} \\circ f_{2009} \\circ \\cdots \\circ f_{1}$ is constant? Note: a function $g$ is constant if $g(a)=g(b)$ for all $a, b$ in the domain of $g$.",
        "answer": "$4^{2010}-2^{2010}$",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1175,
        "difficulty": 5.0
      },
      {
        "id": "d82b92b03359e9f75e938b939b797d02ebaa2eba3ab95cd2495084222110ed78",
        "domain": "omni",
        "question": "A set of positive integers is called [i]fragrant[/i] if it contains at least two elements and each of its elements has a prime factor in common with at least one of the other elements.  Let $P(n)=n^2+n+1$.  What is the least possible positive integer value of $b$ such that there exists a non-negative integer $a$ for which the set $$\\{P(a+1),P(a+2),\\ldots,P(a+b)\\}$$ is fragrant?",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "imo",
        "source_index": 3813,
        "difficulty": 8.0
      },
      {
        "id": "f77b8846fd363aaac515f9b2746b2b497f485a2aa729795cb80d25c5abea1e8d",
        "domain": "omni",
        "question": "The set $S$ consists of 9 distinct positive integers. The average of the two smallest integers in $S$ is 5. The average of the two largest integers in $S$ is 22. What is the greatest possible average of all of the integers of $S$?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 3150,
        "difficulty": 2.5
      },
      {
        "id": "13aaee4c771e23ac48621d478eaadbef4c4807015892b85b50f37c5dac4ad7b7",
        "domain": "omni",
        "question": "How many multiples of 7 between $10^{6}$ and $10^{9}$ are perfect squares?",
        "answer": "4375",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "HMMT_2",
        "source_index": 628,
        "difficulty": 3.5
      },
      {
        "id": "728b759b19efa4744f9ff8a69bf48c7c489f944b08768e2507b2c48413a5b676",
        "domain": "omni",
        "question": "In $\\triangle PQR, \\angle RPQ=90^{\\circ}$ and $S$ is on $PQ$. If $SQ=14, SP=18$, and $SR=30$, what is the area of $\\triangle QRS$?",
        "answer": "168",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "pascal",
        "source_index": 2900,
        "difficulty": 1.5
      },
      {
        "id": "18a7d2bf5cc9ffc0dd0fda4c109d8ea2605e5526cd6fc13e7711825916cf3154",
        "domain": "omni",
        "question": "Determine the smallest positive integer $A$ with an odd number of digits and this property, that both $A$ and the number $B$ created by removing the middle digit of the number $A$ are divisible by $2018$.",
        "answer": "100902018",
        "topic": "Mathematics -> Number Theory -> Divisibility -> Other; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "czech-polish-slovak matches",
        "source_index": 3699,
        "difficulty": 6.0
      },
      {
        "id": "d0a29d1f2c1f19921f463298cc8569965030329d8c72da8376e7e7a9191e0fe3",
        "domain": "omni",
        "question": "A regular hexagon $A B C D E F$ has side length 1 and center $O$. Parabolas $P_{1}, P_{2}, \\ldots, P_{6}$ are constructed with common focus $O$ and directrices $A B, B C, C D, D E, E F, F A$ respectively. Let $\\chi$ be the set of all distinct points on the plane that lie on at least two of the six parabolas. Compute $$\\sum_{X \\in \\chi}|O X|$$ (Recall that the focus is the point and the directrix is the line such that the parabola is the locus of points that are equidistant from the focus and the directrix.)",
        "answer": "35 \\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 835,
        "difficulty": 5.25
      },
      {
        "id": "a3532ff13e040c5979f1bf6e90036390e622ff111a154569c2516e6097491d2e",
        "domain": "omni",
        "question": "Six men and their wives are sitting at a round table with 12 seats. These men and women are very jealous - no man will allow his wife to sit next to any man except for himself, and no woman will allow her husband to sit next to any woman except for herself. In how many distinct ways can these 12 people be seated such that these conditions are satisfied?",
        "answer": "288000",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1733,
        "difficulty": 5.25
      },
      {
        "id": "0d2f3fb24cec100ad4dd4a1ff417bef5fa656525120eace94510e55afa776b1b",
        "domain": "omni",
        "question": "Suppose $a$ and $b$ are positive integers for which $8 a^{a} b^{b}=27 a^{b} b^{a}$. Find $a^{2}+b^{2}$.",
        "answer": "117",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 2642,
        "difficulty": 4.0
      },
      {
        "id": "36f3510288750da0045b68d1ff5d90782dadfd608d4bee03b3dca2b44f94c5e0",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=9, B C=10$, and $C A=17$. Let $B^{\\prime}$ be the reflection of the point $B$ over the line $C A$. Let $G$ be the centroid of triangle $A B C$, and let $G^{\\prime}$ be the centroid of triangle $A B^{\\prime} C$. Determine the length of segment $G G^{\\prime}$.",
        "answer": "\\frac{48}{17}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2023,
        "difficulty": 5.0
      },
      {
        "id": "4a8c541813ea4bbbdc2751f70f55a7469c95a6746b18695f0ff4c1f40608c92f",
        "domain": "omni",
        "question": "Let $T$ be the set of ordered triples $(x,y,z)$, where $x,y,z$ are integers with $0\\leq x,y,z\\leq9$. Players $A$ and $B$ play the following guessing game. Player $A$ chooses a triple $(x,y,z)$ in $T$, and Player $B$ has to discover $A$[i]'s[/i] triple in as few moves as possible.  A [i]move[/i] consists of the following: $B$ gives $A$ a triple $(a,b,c)$ in $T$, and $A$ replies by giving $B$ the number $\\left|x+y-a-b\\right |+\\left|y+z-b-c\\right|+\\left|z+x-c-a\\right|$. Find the minimum number of moves that $B$ needs to be sure of determining $A$[i]'s[/i]  triple.",
        "answer": "3",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "imo_shortlist",
        "source_index": 4149,
        "difficulty": 8.0
      },
      {
        "id": "8c599070100b532a651bd6b8ed777979b3ec08296d6f5d10fb81281ee914be31",
        "domain": "omni",
        "question": "Consider a circular cone with vertex $V$, and let $ABC$ be a triangle inscribed in the base of the cone, such that $AB$ is a diameter and $AC=BC$. Let $L$ be a point on $BV$ such that the volume of the cone is 4 times the volume of the tetrahedron $ABCL$. Find the value of $BL/LV$.",
        "answer": "\\frac{\\pi}{4-\\pi}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 485,
        "difficulty": 5.25
      },
      {
        "id": "5de3ebf98a03ef981f457f0d082ac4b78c6a3c36f60fa27030b6901a5216b054",
        "domain": "omni",
        "question": "Let $r_{1}, r_{2}, \\ldots, r_{7}$ be the distinct complex roots of the polynomial $P(x)=x^{7}-7$. Let $$K=\\prod_{1 \\leq i<j \\leq 7}\\left(r_{i}+r_{j}\\right)$$ that is, the product of all numbers of the form $r_{i}+r_{j}$, where $i$ and $j$ are integers for which $1 \\leq i<j \\leq 7$. Determine the value of $K^{2}$.",
        "answer": "117649",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_11",
        "source_index": 2543,
        "difficulty": 4.0
      },
      {
        "id": "b06ad13307a46101c0750971158aef2493897f7097d3d214a9c96af398ea3078",
        "domain": "omni",
        "question": "Let $f(x)=x^{3}+3 x-1$ have roots $a, b, c$. Given that $$\\frac{1}{a^{3}+b^{3}}+\\frac{1}{b^{3}+c^{3}}+\\frac{1}{c^{3}+a^{3}}$$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$, find $100 m+n$.",
        "answer": "3989",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2268,
        "difficulty": 5.0
      },
      {
        "id": "a42f69d0b2c4062b016a41ec5de51feef198c0c23713cb50ed83ca7c8512551c",
        "domain": "omni",
        "question": "Two different numbers are randomly selected from the set $\\{-3, -1, 0, 2, 4\\}$ and then multiplied together. What is the probability that the product of the two numbers chosen is 0?",
        "answer": "\\frac{2}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3135,
        "difficulty": 2.0
      },
      {
        "id": "e51ce5084bb9b5e17f14735abbd76c0915432e33cd99f67db133d87687f01268",
        "domain": "omni",
        "question": "Let $a \\star b=ab-2$. Compute the remainder when $(((579 \\star 569) \\star 559) \\star \\cdots \\star 19) \\star 9$ is divided by 100.",
        "answer": "29",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 693,
        "difficulty": 5.25
      },
      {
        "id": "e552fdf12d6a32100aa08657cd5b89d2a5b5032ad359c86d677168ba78e39f3d",
        "domain": "omni",
        "question": "Consider parallelogram $A B C D$ with $A B>B C$. Point $E$ on $\\overline{A B}$ and point $F$ on $\\overline{C D}$ are marked such that there exists a circle $\\omega_{1}$ passing through $A, D, E, F$ and a circle $\\omega_{2}$ passing through $B, C, E, F$. If $\\omega_{1}, \\omega_{2}$ partition $\\overline{B D}$ into segments $\\overline{B X}, \\overline{X Y}, \\overline{Y D}$ in that order, with lengths $200,9,80$, respectively, compute $B C$.",
        "answer": "51",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2009,
        "difficulty": 4.5
      },
      {
        "id": "b63375b69042ca0c10a437517cc983976df7ede6e21811462ba877ca17b05a02",
        "domain": "omni",
        "question": "A path of length $n$ is a sequence of points $\\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right), \\ldots,\\left(x_{n}, y_{n}\\right)$ with integer coordinates such that for all $i$ between 1 and $n-1$ inclusive, either (1) $x_{i+1}=x_{i}+1$ and $y_{i+1}=y_{i}$ (in which case we say the $i$th step is rightward) or (2) $x_{i+1}=x_{i}$ and $y_{i+1}=y_{i}+1$ (in which case we say that the $i$th step is upward). This path is said to start at $\\left(x_{1}, y_{1}\\right)$ and end at $\\left(x_{n}, y_{n}\\right)$. Let $P(a, b)$, for $a$ and $b$ nonnegative integers, be the number of paths that start at $(0,0)$ and end at $(a, b)$. Find $\\sum_{i=0}^{10} P(i, 10-i)$.",
        "answer": "1024",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 893,
        "difficulty": 3.5
      },
      {
        "id": "73bc6158580b1c1a5761b5824a5c6928d35405b3e0123ac7c72e633ea6ef169b",
        "domain": "omni",
        "question": "Elena earns $\\$ 13.25$ per hour working at a store. How much does Elena earn in 4 hours?",
        "answer": "\\$53.00",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3472,
        "difficulty": 1.5
      },
      {
        "id": "a48778922e5f5bb7e41bab971899811f177a5f56fda42e46a826b4170e34b4ec",
        "domain": "omni",
        "question": "Yang has the sequence of integers $1,2, \\ldots, 2017$. He makes 2016 swaps in order, where a swap changes the positions of two integers in the sequence. His goal is to end with $2,3, \\ldots, 2017,1$. How many different sequences of swaps can Yang do to achieve his goal?",
        "answer": "2017^{2015}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 705,
        "difficulty": 5.25
      },
      {
        "id": "062c2b9eff213d28adfd1563b4fe571ce3ffeb07acaa77b2a237e2b95a638207",
        "domain": "omni",
        "question": "Each one of 2009 distinct points in the plane is coloured in blue or red, so that on every blue-centered unit circle there are exactly two red points. Find the gratest possible number of blue points.",
        "answer": "45",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "junior_balkan_mo",
        "source_index": 3725,
        "difficulty": 5.0
      },
      {
        "id": "37b1496dda7f7056d22ee65beb70171e210e2be8d7b3b6608f659b61d9ae81ed",
        "domain": "omni",
        "question": "What is the 18 th digit after the decimal point of $\\frac{10000}{9899}$ ?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1023,
        "difficulty": 4.5
      },
      {
        "id": "129be223e2aaeab3e8a15175ad8e04e64493c425ce2f2bde6d6903bc2c69f57b",
        "domain": "omni",
        "question": "A mass of 15 kg is halfway between 10 kg and 20 kg on the vertical axis. What is the age of the cod when its mass is 15 kg?",
        "answer": "7",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3008,
        "difficulty": 1.5
      },
      {
        "id": "feab322a7e9fac42b7fbdb43877ee2d338df9b6ed3c054d1ba2a1ac689f42e5e",
        "domain": "omni",
        "question": "Let $a$ and $b$ be real numbers, and let $r, s$, and $t$ be the roots of $f(x)=x^{3}+a x^{2}+b x-1$. Also, $g(x)=x^{3}+m x^{2}+n x+p$ has roots $r^{2}, s^{2}$, and $t^{2}$. If $g(-1)=-5$, find the maximum possible value of $b$.",
        "answer": "1+\\sqrt{5}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1784,
        "difficulty": 5.0
      },
      {
        "id": "0a71385846888cc1af57de905fcd221df9b0947f072603ab44ae0d39dcea2ea2",
        "domain": "omni",
        "question": "Let $$\\begin{aligned} & A=(1+2 \\sqrt{2}+3 \\sqrt{3}+6 \\sqrt{6})(2+6 \\sqrt{2}+\\sqrt{3}+3 \\sqrt{6})(3+\\sqrt{2}+6 \\sqrt{3}+2 \\sqrt{6})(6+3 \\sqrt{2}+2 \\sqrt{3}+\\sqrt{6}) \\\\ & B=(1+3 \\sqrt{2}+2 \\sqrt{3}+6 \\sqrt{6})(2+\\sqrt{2}+6 \\sqrt{3}+3 \\sqrt{6})(3+6 \\sqrt{2}+\\sqrt{3}+2 \\sqrt{6})(6+2 \\sqrt{2}+3 \\sqrt{3}+\\sqrt{6}) \\end{aligned}$$ Compute the value of $A / B$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 814,
        "difficulty": 5.0
      },
      {
        "id": "65dde927de873b0b396553cdaca3660a77c7ee60f0d866948c8806b6368817b5",
        "domain": "omni",
        "question": "Gustave has 15 steel bars of masses $1 \\mathrm{~kg}, 2 \\mathrm{~kg}, 3 \\mathrm{~kg}, \\ldots, 14 \\mathrm{~kg}, 15 \\mathrm{~kg}$. He also has 3 bags labelled $A, B, C$. He places two steel bars in each bag so that the total mass in each bag is equal to $M \\mathrm{~kg}$. How many different values of $M$ are possible?",
        "answer": "19",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2825,
        "difficulty": 2.5
      },
      {
        "id": "b240cfb949b4973902719c07f866a3f6c2d225f56571b9dda1fbe0b38e022283",
        "domain": "omni",
        "question": "Find the number of ordered triples of integers $(a, b, c)$ with $1 \\leq a, b, c \\leq 100$ and $a^{2} b+b^{2} c+c^{2} a=a b^{2}+b c^{2}+c a^{2}$",
        "answer": "29800",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2316,
        "difficulty": 5.0
      },
      {
        "id": "9a164640db6a2bb687fbfc50d31a953b04cc9e5ef37775d0f1d8144d2938f08b",
        "domain": "omni",
        "question": "Let \\(ABCDEF\\) be a regular hexagon and let point \\(O\\) be the center of the hexagon. How many ways can you color these seven points either red or blue such that there doesn't exist any equilateral triangle with vertices of all the same color?",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2402,
        "difficulty": 4.5
      },
      {
        "id": "41ec0458ebf65a315f815dcc61c2c96e777567beffc833b41636cc4e6f5fea18",
        "domain": "omni",
        "question": "Alicia starts a sequence with $m=3$. What is the fifth term of her sequence following the algorithm: Step 1: Alicia writes down the number $m$ as the first term. Step 2: If $m$ is even, Alicia sets $n=\frac{1}{2} m$. If $m$ is odd, Alicia sets $n=m+1$. Step 3: Alicia writes down the number $m+n+1$ as the next term. Step 4: Alicia sets $m$ equal to the value of the term that she just wrote down in Step 3. Step 5: Alicia repeats Steps 2, 3, 4 until she has five terms, at which point she stops.",
        "answer": "43",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3064,
        "difficulty": 2.0
      },
      {
        "id": "adbbfb18a930d45b4fd5b7da844391b6b50c1053812c04c3f5dcce382521f47d",
        "domain": "omni",
        "question": "In the figure, if $A E=3, C E=1, B D=C D=2$, and $A B=5$, find $A G$.",
        "answer": "3\\sqrt{66} / 7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 464,
        "difficulty": 5.25
      },
      {
        "id": "bf73b0378118c1e4999eb5cb265f47a70dff015ebbff556a3de3f8e14671cbab",
        "domain": "omni",
        "question": "Find the number of positive integer solutions to $n^{x}+n^{y}=n^{z}$ with $n^{z}<2001$.",
        "answer": "10",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 984,
        "difficulty": 5.0
      },
      {
        "id": "e5bace74b42d22f72a7281c82a1d84770a1b6bcba3bc5d2fe7b34b659a8f33fc",
        "domain": "omni",
        "question": "Find the largest real number $\\lambda$ with the following property: for any positive real numbers $p,q,r,s$ there exists a complex number $z=a+bi$($a,b\\in \\mathbb{R})$  such that $$ |b|\\ge \\lambda |a| \\quad \\text{and} \\quad (pz^3+2qz^2+2rz+s) \\cdot (qz^3+2pz^2+2sz+r) =0.$$",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "china_national_olympiad",
        "source_index": 22,
        "difficulty": 8.5
      },
      {
        "id": "d5418cdeb0174a011f6dcb8dfa2694e6df022cf8a9761af7198f47667b912393",
        "domain": "omni",
        "question": "Cyclic quadrilateral $A B C D$ has side lengths $A B=1, B C=2, C D=3$ and $D A=4$. Points $P$ and $Q$ are the midpoints of $\\overline{B C}$ and $\\overline{D A}$. Compute $P Q^{2}$.",
        "answer": "\\frac{116}{35}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 801,
        "difficulty": 5.0
      },
      {
        "id": "0e1773ceefd28060f7ad7316c73b2cb9ad55319c42d712f54f89e2ac67e7d448",
        "domain": "omni",
        "question": "Let $x_{1}, \\ldots, x_{100}$ be defined so that for each $i, x_{i}$ is a (uniformly) random integer between 1 and 6 inclusive. Find the expected number of integers in the set $\\{x_{1}, x_{1}+x_{2}, \\ldots, x_{1}+x_{2}+\\ldots+x_{100}\\}$ that are multiples of 6.",
        "answer": "\\frac{50}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 882,
        "difficulty": 4.0
      },
      {
        "id": "3df10c07e23f9b88092f1f88ee134f0ea45bfeea3df29108896dc71fe3b6ea85",
        "domain": "omni",
        "question": "What is the largest real number $\\theta$ less than $\\pi$ (i.e. $\\theta<\\pi$ ) such that $\\prod_{k=0}^{10} \\cos \\left(2^{k} \\theta\\right) \\neq 0$ and $\\prod_{k=0}^{10}\\left(1+\\frac{1}{\\cos \\left(2^{k} \\theta\\right)}\\right)=1 ?",
        "answer": "\\frac{2046 \\pi}{2047}",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "HMMT_2",
        "source_index": 889,
        "difficulty": 5.25
      },
      {
        "id": "89e082201ea9d0080b0ee9d1012caa1e8ac6f8a78569d5fce8c40baf8ea73b3a",
        "domain": "omni",
        "question": "Let $P$ be a polynomial with integer coefficients such that $P(0)+P(90)=2018$. Find the least possible value for $|P(20)+P(70)|$.",
        "answer": "\\[ 782 \\]",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 314,
        "difficulty": 5.0
      },
      {
        "id": "ccc5236a50b9d986fe769f76b98dd886444b0d0cefe2c9f1c9be3795d522e8cf",
        "domain": "omni",
        "question": "The pairwise greatest common divisors of five positive integers are $2,3,4,5,6,7,8, p, q, r$ in some order, for some positive integers $p, q, r$. Compute the minimum possible value of $p+q+r$.",
        "answer": "9",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 1919,
        "difficulty": 5.0
      },
      {
        "id": "65b8f8bc81340284ec83a1a6226b79c1acb1026ff50f58dda14403d447dff813",
        "domain": "omni",
        "question": "Bob is coloring lattice points in the coordinate plane. Find the number of ways Bob can color five points in $\\{(x, y) \\mid 1 \\leq x, y \\leq 5\\}$ blue such that the distance between any two blue points is not an integer.",
        "answer": "80",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 857,
        "difficulty": 4.5
      },
      {
        "id": "432a98cb03142a0afa88db857546cc4c7f368bc485788564818bc25e3b8f86ad",
        "domain": "omni",
        "question": "If \\( x=2 \\), what is the value of \\( (x+2-x)(2-x-2) \\)?",
        "answer": "-4",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2812,
        "difficulty": 1.0
      },
      {
        "id": "4acac4d3ea578243e013da02e62fdf43e783506bb212a8357c5a75fde1ba5d1a",
        "domain": "omni",
        "question": "Let $ABC$ be an acute triangle.  Let $DAC,EAB$, and $FBC$ be isosceles triangles exterior to $ABC$, with $DA=DC, EA=EB$, and $FB=FC$, such that\n\\[\n\\angle ADC = 2\\angle BAC, \\quad \\angle BEA= 2 \\angle ABC, \\quad\n\\angle CFB = 2 \\angle ACB.\n\\]\nLet $D'$  be the intersection of lines $DB$ and $EF$, let $E'$ be the intersection of $EC$ and $DF$, and let $F'$   be the intersection of $FA$ and $DE$. Find, with proof, the value of the sum\n\\[\n\\frac{DB}{DD'}+\\frac{EC}{EE'}+\\frac{FA}{FF'}.\n\\]",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "imo_shortlist",
        "source_index": 4222,
        "difficulty": 8.0
      },
      {
        "id": "1a4572778ce3fde02d51f5e40ececf9409b5044f63b0c54d3c7ee650fb20364c",
        "domain": "omni",
        "question": "Suppose a real number \\(x>1\\) satisfies \\(\\log _{2}\\left(\\log _{4} x\\right)+\\log _{4}\\left(\\log _{16} x\\right)+\\log _{16}\\left(\\log _{2} x\\right)=0\\). Compute \\(\\log _{2}\\left(\\log _{16} x\\right)+\\log _{16}\\left(\\log _{4} x\\right)+\\log _{4}\\left(\\log _{2} x\\right)\\).",
        "answer": "-\\frac{1}{4}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 1534,
        "difficulty": 3.5
      },
      {
        "id": "0c8dabbe9a7ac3ddcb8a8181104e6b9aa37fa2c89fd0bca1478763881301adab",
        "domain": "omni",
        "question": "Let $x$ be a real number. Find the maximum value of $2^{x(1-x)}$.",
        "answer": "\\sqrt[4]{2}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "HMMT_11",
        "source_index": 2469,
        "difficulty": 4.0
      },
      {
        "id": "32863ef65912162f171ea1cb3a339a99fab3f0c78bb4d6519139a7e59276c9c8",
        "domain": "omni",
        "question": "Let $a, b, c, d$ be real numbers such that $\\min (20 x+19,19 x+20)=(a x+b)-|c x+d|$ for all real numbers $x$. Find $a b+c d$.",
        "answer": "380",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1930,
        "difficulty": 5.0
      },
      {
        "id": "f607874c3f7c41ae109dae625e0483a19e801b1f4131cfcdd3b8c44d7c718592",
        "domain": "omni",
        "question": "There are 2 runners on the perimeter of a regular hexagon, initially located at adjacent vertices. Every second, each of the runners independently moves either one vertex to the left, with probability $\\frac{1}{2}$, or one vertex to the right, also with probability $\\frac{1}{2}$. Find the probability that after a 2013 second run (in which the runners switch vertices 2013 times each), the runners end up at adjacent vertices once again.",
        "answer": "\\frac{2}{3}+\\frac{1}{3}\\left(\\frac{1}{4}\\right)^{2013}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2128,
        "difficulty": 4.0
      },
      {
        "id": "65c9d736ab1abe21763f19bcc2d918c4a6ab9c86ceba6270b4f49b5473b2c19e",
        "domain": "omni",
        "question": "Barry picks infinitely many points inside a unit circle, each independently and uniformly at random, $P_{1}, P_{2}, \\ldots$ Compute the expected value of $N$, where $N$ is the smallest integer such that $P_{N+1}$ is inside the convex hull formed by the points $P_{1}, P_{2}, \\ldots, P_{N}$. Submit a positive real number $E$. If the correct answer is $A$, you will receive $\\lfloor 100 \\cdot \\max (0.2099-|E-A|, 0)\\rfloor$ points.",
        "answer": "6.54",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 708,
        "difficulty": 5.0
      },
      {
        "id": "f9d53ef62e8dc8ad2f0611a05fc50f77264e59e63f55e65621ea8edd8023a7e2",
        "domain": "omni",
        "question": "The sequence $a_{1}, a_{2}, a_{3}, \\ldots$ of real numbers satisfies the recurrence $a_{n+1}=\\frac{a_{n}^{2}-a_{n-1}+2 a_{n}}{a_{n-1}+1}$. Given that $a_{1}=1$ and $a_{9}=7$, find $a_{5}$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 437,
        "difficulty": 5.0
      },
      {
        "id": "7525f5e8b03d936b31e38c42ccede33a51bd57351dfab8ab3ccca4f22e9bf07b",
        "domain": "omni",
        "question": "A tetrahedron has all its faces triangles with sides $13,14,15$. What is its volume?",
        "answer": "42 \\sqrt{55}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 736,
        "difficulty": 5.0
      },
      {
        "id": "7925c8e54396f8a0d8d89cb8d7b2116f12c773529bef099ce124f974c28562ff",
        "domain": "omni",
        "question": "A lame king is a chess piece that can move from a cell to any cell that shares at least one vertex with it, except for the cells in the same column as the current cell. A lame king is placed in the top-left cell of a $7 \\times 7$ grid. Compute the maximum number of cells it can visit without visiting the same cell twice (including its starting cell).",
        "answer": "43",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1277,
        "difficulty": 5.0
      },
      {
        "id": "8674c4d9d46097bbc7625f2695817658c6945cc77285f6982ec64da075420bc6",
        "domain": "omni",
        "question": "A complex quartic polynomial $Q$ is quirky if it has four distinct roots, one of which is the sum of the other three. There are four complex values of $k$ for which the polynomial $Q(x)=x^{4}-k x^{3}-x^{2}-x-45$ is quirky. Compute the product of these four values of $k$.",
        "answer": "720",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2216,
        "difficulty": 4.5
      },
      {
        "id": "55c5245181ff936a708b766fd93e9ebd5d9cc8d47d62a00698d4e3109e65b380",
        "domain": "omni",
        "question": "The function $f$ satisfies $f(x)+f(2 x+y)+5 x y=f(3 x-y)+2 x^{2}+1$ for all real numbers $x, y$. Determine the value of $f(10)$.",
        "answer": "-49",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1217,
        "difficulty": 5.5
      },
      {
        "id": "07fe331d971066afafdb0a84cad2a4e2b65d113672a3e33af1c8fa838d62ab40",
        "domain": "omni",
        "question": "What is the value of the expression $\frac{3}{10}+\frac{3}{100}+\frac{3}{1000}$?",
        "answer": "0.333",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3014,
        "difficulty": 1.0
      },
      {
        "id": "60d27405152881ea92938da39d71f4b794dd2de0ca3850d51908c512325d22a6",
        "domain": "omni",
        "question": "Nine fair coins are flipped independently and placed in the cells of a 3 by 3 square grid. Let $p$ be the probability that no row has all its coins showing heads and no column has all its coins showing tails. If $p=\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
        "answer": "8956",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2211,
        "difficulty": 4.0
      },
      {
        "id": "b9becf52434b7dfababf8b60e0ae2a1d6b5481a44ac00e8e283604a6548313b5",
        "domain": "omni",
        "question": "Compute $$\\sum_{n=0}^{\\infty} \\frac{n}{n^{4}+n^{2}+1}$$",
        "answer": "1/2",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable; Mathematics -> Algebra -> Algebraic Expressions -> Other",
        "source": "HMMT_2",
        "source_index": 1540,
        "difficulty": 5.0
      },
      {
        "id": "8fcf36ec913229ce0c568924777e89586d1611ec0afaab969274e85ee379f6db",
        "domain": "omni",
        "question": "Three $1 \times 1 \times 1$ cubes are joined side by side. What is the surface area of the resulting prism?",
        "answer": "14",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Surface Area",
        "source": "pascal",
        "source_index": 3098,
        "difficulty": 1.0
      },
      {
        "id": "a6ac1cece62ac25ff1ed7c50c0453d933a46e19b8e02fdd3f03b7909c1fd9a44",
        "domain": "omni",
        "question": "What is the greatest possible value of $n$ if Juliana chooses three different numbers from the set $\\{-6,-4,-2,0,1,3,5,7\\}$ and multiplies them together to obtain the integer $n$?",
        "answer": "168",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3061,
        "difficulty": 2.5
      },
      {
        "id": "877172a63827802a615b05c89ddb64b09914fa634e8aee8bf4887f167d3230c1",
        "domain": "omni",
        "question": "The taxicab distance between points $\\left(x_{1}, y_{1}\\right)$ and $\\left(x_{2}, y_{2}\\right)$ is $\\left|x_{2}-x_{1}\\right|+\\left|y_{2}-y_{1}\\right|$. A regular octagon is positioned in the $x y$ plane so that one of its sides has endpoints $(0,0)$ and $(1,0)$. Let $S$ be the set of all points inside the octagon whose taxicab distance from some octagon vertex is at most \\frac{2}{3}$. The area of $S$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "2309",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2235,
        "difficulty": 5.0
      },
      {
        "id": "0d9b6311424994dca219b2510e00701b7f3bcfc380b7a2c7b0a5f4e40b551a10",
        "domain": "omni",
        "question": "How many sequences of integers $(a_{1}, \\ldots, a_{7})$ are there for which $-1 \\leq a_{i} \\leq 1$ for every $i$, and $a_{1} a_{2}+a_{2} a_{3}+a_{3} a_{4}+a_{4} a_{5}+a_{5} a_{6}+a_{6} a_{7}=4$?",
        "answer": "38",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1874,
        "difficulty": 5.0
      },
      {
        "id": "7e78aa2d65b617ee211e7b7ee0bed4509bfd11ff5d21e9c761d5edd8adcc8603",
        "domain": "omni",
        "question": "Compute the sum of all positive real numbers \\(x \\leq 5\\) satisfying \\(x=\\frac{\\left\\lceil x^{2}\\right\\rceil+\\lceil x\\rceil \\cdot\\lfloor x\\rfloor}{\\lceil x\\rceil+\\lfloor x\\rfloor}\\).",
        "answer": "85",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2186,
        "difficulty": 6.25
      },
      {
        "id": "f96ab9830527521bb3e5c8e38e190e6a7774bb07b12a1c57c9cc0ff209a820c9",
        "domain": "omni",
        "question": "Let $a, b, c$ be not necessarily distinct integers between 1 and 2011, inclusive. Find the smallest possible value of $\\frac{a b+c}{a+b+c}$.",
        "answer": "$\\frac{2}{3}$",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1959,
        "difficulty": 4.0
      },
      {
        "id": "a4350c91cc4c5085c2727e163656c0b19cc19689a1666829c6f036d112d492b9",
        "domain": "omni",
        "question": "We call a positive integer $t$ good if there is a sequence $a_{0}, a_{1}, \\ldots$ of positive integers satisfying $a_{0}=15, a_{1}=t$, and $a_{n-1} a_{n+1}=\\left(a_{n}-1\\right)\\left(a_{n}+1\\right)$ for all positive integers $n$. Find the sum of all good numbers.",
        "answer": "296",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 535,
        "difficulty": 4.0
      },
      {
        "id": "5b704db735a63cfb2a0098a6eccf3965f9deed48ec19658f9ecc042857c921a4",
        "domain": "omni",
        "question": "Let $ f(n)$ be a function defined on the set of all positive integers and having its values in the same set. Suppose that $ f(f(n) \\plus{} f(m)) \\equal{} m \\plus{} n$ for all positive integers $ n,m.$ Find the possible value for $ f(1988).$",
        "answer": "1988",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "imo_shortlist",
        "source_index": 3876,
        "difficulty": 5.5
      },
      {
        "id": "71cde0705c45fb6729e91688566d7e9b3f04dc3e5c168c7fbda4f0a086e849d6",
        "domain": "omni",
        "question": "If $x$ and $y$ are positive real numbers with $\\frac{1}{x+y}=\\frac{1}{x}-\\frac{1}{y}$, what is the value of $\\left(\\frac{x}{y}+\\frac{y}{x}\\right)^{2}$?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2836,
        "difficulty": 2.5
      },
      {
        "id": "5c9125991a281195085b9f61d6eec1b336986c6a14b0124817a652dba440dbd1",
        "domain": "omni",
        "question": "The country of HMMTLand has 8 cities. Its government decides to construct several two-way roads between pairs of distinct cities. After they finish construction, it turns out that each city can reach exactly 3 other cities via a single road, and from any pair of distinct cities, either exactly 0 or 2 other cities can be reached from both cities by a single road. Compute the number of ways HMMTLand could have constructed the roads.",
        "answer": "875",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 1137,
        "difficulty": 5.0
      },
      {
        "id": "cf598d678199aae4fe2e6e5b5605986d5eac9c73fc479e0b60a2ce7aee0e2f08",
        "domain": "omni",
        "question": "For how many ordered triplets $(a, b, c)$ of positive integers less than 10 is the product $a \\times b \\times c$ divisible by 20?",
        "answer": "102",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1185,
        "difficulty": 4.0
      },
      {
        "id": "254627926bc8d303d58b9bd5050bdba78154e824b6e0ddcd612e62c2455d99b5",
        "domain": "omni",
        "question": "A collection $\\mathcal{S}$ of 10000 points is formed by picking each point uniformly at random inside a circle of radius 1. Let $N$ be the expected number of points of $\\mathcal{S}$ which are vertices of the convex hull of the $\\mathcal{S}$. (The convex hull is the smallest convex polygon containing every point of $\\mathcal{S}$.) Estimate $N$.",
        "answer": "72.8",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 532,
        "difficulty": 5.0
      },
      {
        "id": "20266a5fb940352676a307d7fe98bc416e51a660d211773b5d31864ab9bc9e77",
        "domain": "omni",
        "question": "Let $A B C$ be an acute triangle with $A$-excircle $\\Gamma$. Let the line through $A$ perpendicular to $B C$ intersect $B C$ at $D$ and intersect $\\Gamma$ at $E$ and $F$. Suppose that $A D=D E=E F$. If the maximum value of $\\sin B$ can be expressed as $\\frac{\\sqrt{a}+\\sqrt{b}}{c}$ for positive integers $a, b$, and $c$, compute the minimum possible value of $a+b+c$.",
        "answer": "705",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1407,
        "difficulty": 5.25
      },
      {
        "id": "9d79b41c5d705ae557be43ab71e9f3701b5e1170847d56f8339934ee8b53bc7f",
        "domain": "omni",
        "question": "Find the smallest positive integer $b$ such that $1111_{b}$ ( 1111 in base $b$) is a perfect square. If no such $b$ exists, write \"No solution\".",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1748,
        "difficulty": 4.5
      },
      {
        "id": "e75af83ce78b11237cf1c9fc2e10410c7d81ef1d0eb799b6faa2e638543dcd67",
        "domain": "omni",
        "question": "Suppose $P(x)$ is a polynomial such that $P(1)=1$ and $$\\frac{P(2 x)}{P(x+1)}=8-\\frac{56}{x+7}$$ for all real $x$ for which both sides are defined. Find $P(-1)$.",
        "answer": "-5/21",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1348,
        "difficulty": 5.0
      },
      {
        "id": "f75ab858eff1749c03b037ffab069bb5069fa469c1b9d1abcc37937900c38e93",
        "domain": "omni",
        "question": "For positive integers $m, n$, let \\operatorname{gcd}(m, n) denote the largest positive integer that is a factor of both $m$ and $n$. Compute $$\\sum_{n=1}^{91} \\operatorname{gcd}(n, 91)$$",
        "answer": "325",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 1952,
        "difficulty": 3.5
      },
      {
        "id": "477d7fe3d076eac7e3408884a16b6de3dd1ab49811623cc591a67a13774004e9",
        "domain": "omni",
        "question": "Let $ABCD$ be a square with side length $1$. How many points $P$ inside the square (not on its sides) have the property that the square can be cut into $10$ triangles of equal area such that all of them have $P$ as a vertex?",
        "answer": "16",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "th_igo",
        "source_index": 3687,
        "difficulty": 6.5
      },
      {
        "id": "895d54ed69cbb3781e423dd5afc0465ba99a5e4d910f892670a1f74318e667be",
        "domain": "omni",
        "question": "Call a positive integer $N \\geq 2$ \"special\" if for every $k$ such that $2 \\leq k \\leq N, N$ can be expressed as a sum of $k$ positive integers that are relatively prime to $N$ (although not necessarily relatively prime to each other). How many special integers are there less than $100$?",
        "answer": "50",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1321,
        "difficulty": 4.0
      },
      {
        "id": "c1074f3fa3c8a57d479a494ef7ba003a1ec313628d60896b9c8c4212df5d8497",
        "domain": "omni",
        "question": "Five equally skilled tennis players named Allen, Bob, Catheryn, David, and Evan play in a round robin tournament, such that each pair of people play exactly once, and there are no ties. In each of the ten games, the two players both have a $50 \\%$ chance of winning, and the results of the games are independent. Compute the probability that there exist four distinct players $P_{1}, P_{2}, P_{3}, P_{4}$ such that $P_{i}$ beats $P_{i+1}$ for $i=1,2,3,4$. (We denote $P_{5}=P_{1}$ ).",
        "answer": "\\frac{49}{64}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2120,
        "difficulty": 4.0
      },
      {
        "id": "1a90f0d5c12f7ad5c1dc3843202f5bd8ebd573345a10c789850d64e647689899",
        "domain": "omni",
        "question": "Let $A B C D$ be a quadrilateral with an inscribed circle $\\omega$ that has center $I$. If $I A=5, I B=7, I C=4, I D=9$, find the value of $\\frac{A B}{C D}$.",
        "answer": "\\frac{35}{36}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2144,
        "difficulty": 3.5
      },
      {
        "id": "d5c018919fa0fa060210760fe60532899760189ad73d7470f54f9a2dbd352a03",
        "domain": "omni",
        "question": "Let $ABC$ be a triangle with circumcenter $O$ such that $AC=7$. Suppose that the circumcircle of $AOC$ is tangent to $BC$ at $C$ and intersects the line $AB$ at $A$ and $F$. Let $FO$ intersect $BC$ at $E$. Compute $BE$.",
        "answer": "\\frac{7}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1364,
        "difficulty": 6.0
      },
      {
        "id": "8c27c5913f84ebeea39481bee44272f5829e1dd02102ecabf2622548b2b85992",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that $\\frac{5^{n+1}+2^{n+1}}{5^{n}+2^{n}}>4.99$.",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 580,
        "difficulty": 5.0
      },
      {
        "id": "bd66a6b2faaa93c56063e6870ef5574a6114e543ebe550242899e5cef90905a3",
        "domain": "omni",
        "question": "Let $T$ be the set of numbers of the form $2^{a} 3^{b}$ where $a$ and $b$ are integers satisfying $0 \\leq a, b \\leq 5$. How many subsets $S$ of $T$ have the property that if $n$ is in $S$ then all positive integer divisors of $n$ are in $S$ ?",
        "answer": "924",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2480,
        "difficulty": 5.0
      },
      {
        "id": "5c89daac0ccf1eddfa28eaaa4fcd5ab2f4cab52959b33e4c13c54ce39cd932a8",
        "domain": "omni",
        "question": "How many ordered pairs $(a, b)$ of positive integers satisfy $a^{2}+b^{2}=50$?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "cayley",
        "source_index": 2673,
        "difficulty": 2.5
      },
      {
        "id": "b437e2128d5c2ee9eec199d2cf89cf71fad47eb0e4d0c881dc24d98ddac54d9a",
        "domain": "omni",
        "question": "A positive integer $n$ is $inverosimil$ if there exists $n$ integers not necessarily distinct such that the sum and the product of this integers are equal to $n$. How many positive integers less than or equal to $2022$ are $inverosimils$?",
        "answer": "1010",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "centroamerican_and_caribbean_math_olympiad",
        "source_index": 3942,
        "difficulty": 6.0
      },
      {
        "id": "cdb7c465c3f76a941532657eefc6202807e43311370344effee43a3477c907b7",
        "domain": "omni",
        "question": "Franklin has four bags, numbered 1 through 4. Initially, the first bag contains fifteen balls, numbered 1 through 15 , and the other bags are empty. Franklin randomly pulls a pair of balls out of the first bag, throws away the ball with the lower number, and moves the ball with the higher number into the second bag. He does this until there is only one ball left in the first bag. He then repeats this process in the second and third bag until there is exactly one ball in each bag. What is the probability that ball 14 is in one of the bags at the end?",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 749,
        "difficulty": 5.0
      },
      {
        "id": "2a02cfdd43e9fa9a5bbb9d75de7f6e61d3c517f49324ad9c8164fe265f80bfe5",
        "domain": "omni",
        "question": "Suppose $a, b, c, d$, and $e$ are objects that we can multiply together, but the multiplication doesn't necessarily satisfy the associative law, i.e. ( $x y) z$ does not necessarily equal $x(y z)$. How many different ways are there to interpret the product abcde?",
        "answer": "14",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 404,
        "difficulty": 5.0
      },
      {
        "id": "35aadca35fd0e1a42e76dcb1c38989457765a18017646c72931ce010af4990de",
        "domain": "omni",
        "question": "Two concentric circles have radii $r$ and $R>r$. Three new circles are drawn so that they are each tangent to the big two circles and tangent to the other two new circles. Find $\\frac{R}{r}$.",
        "answer": "3",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 378,
        "difficulty": 3.5
      },
      {
        "id": "bfe674a38f50b8fd74a53918aed77ebc55d17dd12af22a393999af756f242b48",
        "domain": "omni",
        "question": "What is the value of $\frac{8+4}{8-4}$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2874,
        "difficulty": 1.5
      },
      {
        "id": "c9fa06ec0d2d3c2e1d4e66ea3789078860b94f59d03985d52d296c5d303803f0",
        "domain": "omni",
        "question": "An integer $n$ is decreased by 2 and then multiplied by 5. If the result is 85, what is the value of $n$?",
        "answer": "19",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3200,
        "difficulty": 1.0
      },
      {
        "id": "448a610930bbcf1dbb8f414deb86454150da11fdbc054f495943d5a0e9c68005",
        "domain": "omni",
        "question": "If $x=18$ is one of the solutions of the equation $x^{2}+12x+c=0$, what is the other solution of this equation?",
        "answer": "-30",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "fermat",
        "source_index": 2716,
        "difficulty": 2.5
      },
      {
        "id": "909f56f94198db70840f96a3623119e1dfda65df1dcfaa964fbfc1a69d0294c6",
        "domain": "omni",
        "question": "If $a(x+b)=3 x+12$ for all values of $x$, what is the value of $a+b$?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2682,
        "difficulty": 1.5
      },
      {
        "id": "289ee91954f64628e89b2bb6b3f6bd43bd14ab351080a401c5639e6bcb54649c",
        "domain": "omni",
        "question": "The Cayley Corner Store sells three types of toys: Exes, Wyes and Zeds. All Exes are identical, all Wyes are identical, and all Zeds are identical. The mass of 2 Exes equals the mass of 29 Wyes. The mass of 1 Zed equals the mass of 16 Exes. The mass of 1 Zed equals the mass of how many Wyes?",
        "answer": "232",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2691,
        "difficulty": 1.5
      },
      {
        "id": "d9df7b10c2ed94c0566cf17b6021ce1b63a024f896034f666cadba2fa96607ad",
        "domain": "omni",
        "question": "Reimu has 2019 coins $C_{0}, C_{1}, \\ldots, C_{2018}$, one of which is fake, though they look identical to each other (so each of them is equally likely to be fake). She has a machine that takes any two coins and picks one that is not fake. If both coins are not fake, the machine picks one uniformly at random. For each $i=1,2, \\ldots, 1009$, she puts $C_{0}$ and $C_{i}$ into the machine once, and machine picks $C_{i}$. What is the probability that $C_{0}$ is fake?",
        "answer": "\\frac{2^{1009}}{2^{1009}+1009}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 825,
        "difficulty": 4.0
      },
      {
        "id": "83f245f2743310e3e3ea70b9a8276bd9ae9bcb6ef2d93691afbec2173f2ad2d2",
        "domain": "omni",
        "question": "In the $5 \\times 5$ grid shown, 15 cells contain X's and 10 cells are empty. What is the smallest number of X's that must be moved so that each row and each column contains exactly three X's?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "fermat",
        "source_index": 3030,
        "difficulty": 2.0
      },
      {
        "id": "a67b0811b87ffbc0b8f42c2610bdebd889f70d8039c7b5bdc7da6683b6b81fdf",
        "domain": "omni",
        "question": "A function $f:\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ is said to be nasty if there do not exist distinct $a, b \\in\\{1,2,3,4,5\\}$ satisfying $f(a)=b$ and $f(b)=a$. How many nasty functions are there?",
        "answer": "1950",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1953,
        "difficulty": 4.5
      },
      {
        "id": "8800cf26493f29a284307ec8a2d2b00c099dd27ee1aa238147f6410d5a87dd75",
        "domain": "omni",
        "question": "A polynomial $P$ has four roots, $\\frac{1}{4}, \\frac{1}{2}, 2,4$. The product of the roots is 1, and $P(1)=1$. Find $P(0)$.",
        "answer": "\\frac{8}{9}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1556,
        "difficulty": 5.0
      },
      {
        "id": "891b52d81cc0a64792bca2b17f6f5867fb92c36cf9e435f75455bd2f8fbf6b33",
        "domain": "omni",
        "question": "Let $S_{0}=0$ and let $S_{k}$ equal $a_{1}+2 a_{2}+\\ldots+k a_{k}$ for $k \\geq 1$. Define $a_{i}$ to be 1 if $S_{i-1}<i$ and -1 if $S_{i-1} \\geq i$. What is the largest $k \\leq 2010$ such that $S_{k}=0$?",
        "answer": "1092",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series",
        "source": "HMMT_2",
        "source_index": 1337,
        "difficulty": 5.0
      },
      {
        "id": "4b3796caf0b180d62a6703fc03a12faf32d2a135149785b918dfbe5177f8d0b1",
        "domain": "omni",
        "question": "Find the number of ordered quadruples of positive integers $(a, b, c, d)$ such that $a, b, c$, and $d$ are all (not necessarily distinct) factors of 30 and $abcd>900$.",
        "answer": "1940",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 906,
        "difficulty": 5.0
      },
      {
        "id": "350470bf498e89d837ab87a11bbafc02b34274ed980156353ff13bbc3c883d56",
        "domain": "omni",
        "question": "Given a permutation $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$, define a rotated cycle as a set of three integers $i, j, k$ such that $i<j<k$ and $\\pi(j)<\\pi(k)<\\pi(i)$. What is the total number of rotated cycles over all permutations $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$ ?",
        "answer": "72576000",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2433,
        "difficulty": 5.25
      },
      {
        "id": "ba496a3aac32d2b75141f8a6a7529f167a617c21a784a57e4f37de8373c4b514",
        "domain": "omni",
        "question": "Let $\\mathcal{P}$ be a parabola with focus $F$ and directrix $\\ell$. A line through $F$ intersects $\\mathcal{P}$ at two points $A$ and $B$. Let $D$ and $C$ be the feet of the altitudes from $A$ and $B$ onto $\\ell$, respectively. Given that $AB=20$ and $CD=14$, compute the area of $ABCD$.",
        "answer": "140",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2153,
        "difficulty": 4.5
      },
      {
        "id": "b0232295e418fb8ba4a6f1fb40e2e7bbd18efa72965250189d10fb606d8bbf28",
        "domain": "omni",
        "question": "$A B C$ is an acute triangle with incircle $\\omega$. $\\omega$ is tangent to sides $\\overline{B C}, \\overline{C A}$, and $\\overline{A B}$ at $D, E$, and $F$ respectively. $P$ is a point on the altitude from $A$ such that $\\Gamma$, the circle with diameter $\\overline{A P}$, is tangent to $\\omega$. $\\Gamma$ intersects $\\overline{A C}$ and $\\overline{A B}$ at $X$ and $Y$ respectively. Given $X Y=8, A E=15$, and that the radius of $\\Gamma$ is 5, compute $B D \\cdot D C$.",
        "answer": "\\frac{675}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1025,
        "difficulty": 6.0
      },
      {
        "id": "160292c92c74b266ebf69f0914e17861ab73e3ba05f2c36bd0c03309b3652afe",
        "domain": "omni",
        "question": "Let $f$ be a monic cubic polynomial satisfying $f(x)+f(-x)=0$ for all real numbers $x$. For all real numbers $y$, define $g(y)$ to be the number of distinct real solutions $x$ to the equation $f(f(x))=y$. Suppose that the set of possible values of $g(y)$ over all real numbers $y$ is exactly $\\{1,5,9\\}$. Compute the sum of all possible values of $f(10)$.",
        "answer": "970",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1488,
        "difficulty": 8.0
      },
      {
        "id": "b4f52690f8e12d9397b220305a060724c673a89aa8c4b638d997c999c9940594",
        "domain": "omni",
        "question": "Let $\\triangle ABC$ be an equilateral triangle of side length 1. Let $D,E,F$ be points on $BC,AC,AB$ respectively, such that $\\frac{DE}{20} = \\frac{EF}{22} = \\frac{FD}{38}$. Let $X,Y,Z$ be on lines $BC,CA,AB$ respectively, such that $XY\\perp DE, YZ\\perp EF, ZX\\perp FD$. Find all possible values of $\\frac{1}{[DEF]} + \\frac{1}{[XYZ]}$.",
        "answer": "\\frac{97 \\sqrt{2} + 40 \\sqrt{3}}{15}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "china_national_olympiad",
        "source_index": 81,
        "difficulty": 8.0
      },
      {
        "id": "9bd6c17cfe7b0349a0d7403c22b9c5dc46d8858bca6f93bb65f8d7053bb1470a",
        "domain": "omni",
        "question": "Count how many 8-digit numbers there are that contain exactly four nines as digits.",
        "answer": "433755",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 812,
        "difficulty": 4.0
      },
      {
        "id": "00f340784df767c17a057acdfe50594c5331006a30172d7316e5fcf873b4b7d8",
        "domain": "omni",
        "question": "A four-digit positive integer is called [i]virtual[/i] if it has the form $\\overline{abab}$, where $a$ and $b$ are digits and $a \\neq 0$. For example 2020, 2121 and 2222 are virtual numbers, while 2002 and 0202 are not. Find all virtual numbers of the form $n^2+1$, for some positive integer $n$.",
        "answer": "8282",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "centroamerican_and_caribbean_math_olympiad",
        "source_index": 3585,
        "difficulty": 5.0
      },
      {
        "id": "80424b2e3aba90eec0100854dde54cb7c5756c5f4c790a682af54e160c1e30f8",
        "domain": "omni",
        "question": "Let $a,b,c$ be positive real numbers such that $a+b+c = 3$.  Find the minimum value of the expression \\[A=\\dfrac{2-a^3}a+\\dfrac{2-b^3}b+\\dfrac{2-c^3}c.\\]",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "junior_balkan_mo",
        "source_index": 4097,
        "difficulty": 5.0
      },
      {
        "id": "bf486cb60115c9b8b770177e6a43b5f8e5f62e44d2f6b6ceaa85dd089b72a50a",
        "domain": "omni",
        "question": "An electric car is charged 3 times per week for 52 weeks. The cost to charge the car each time is $0.78. What is the total cost to charge the car over these 52 weeks?",
        "answer": "\\$121.68",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2971,
        "difficulty": 1.5
      },
      {
        "id": "896b6bcc3216284fcc5652e3aa7ca062c010683def8591d6e82f5d58c4e87611",
        "domain": "omni",
        "question": "Triangle $ABC$ has an area 1.  Points $E,F,G$ lie, respectively, on sides $BC$, $CA$, $AB$ such that $AE$ bisects $BF$ at point $R$, $BF$ bisects $CG$ at point $S$, and $CG$ bisects $AE$ at point $T$. Find the area of the triangle $RST$.",
        "answer": "\\frac{7 - 3 \\sqrt{5}}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "putnam",
        "source_index": 3268,
        "difficulty": 8.0
      },
      {
        "id": "b9384324cb58a06ea2e6b6213736014cab2ce4202106d0a8f01596c3c20d1cf9",
        "domain": "omni",
        "question": "Let $A B C D$ be a square of side length 5. A circle passing through $A$ is tangent to segment $C D$ at $T$ and meets $A B$ and $A D$ again at $X \\neq A$ and $Y \\neq A$, respectively. Given that $X Y=6$, compute $A T$.",
        "answer": "\\sqrt{30}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2270,
        "difficulty": 4.0
      },
      {
        "id": "2d6eff154ae7db8cb43089022d8c768c96390c620e74bbc4388b7c15929da485",
        "domain": "omni",
        "question": "Let $f(x)=x^{4}+14 x^{3}+52 x^{2}+56 x+16$. Let $z_{1}, z_{2}, z_{3}, z_{4}$ be the four roots of $f$. Find the smallest possible value of $|z_{a} z_{b}+z_{c} z_{d}|$ where $\\{a, b, c, d\\}=\\{1,2,3,4\\}$.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1249,
        "difficulty": 4.5
      },
      {
        "id": "fb9c6f4a3cf98b7453be2fc21082cde7875fa7279e314e7422127d7b5d9df3b3",
        "domain": "omni",
        "question": "For an integer $x \\geq 1$, let $p(x)$ be the least prime that does not divide $x$, and define $q(x)$ to be the product of all primes less than $p(x)$. In particular, $p(1) = 2.$ For $x$ having $p(x) = 2$, define $q(x) = 1$. Consider the sequence $x_0, x_1, x_2, \\ldots$ defined by $x_0 = 1$ and \\[ x_{n+1} = \\frac{x_n p(x_n)}{q(x_n)} \\] for $n \\geq 0$. Find all $n$ such that $x_n = 1995$.",
        "answer": "142",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Sequences -> Other",
        "source": "imo_shortlist",
        "source_index": 4018,
        "difficulty": 5.5
      },
      {
        "id": "2246cb5220bd0d678ea1623719814e98cb2f23ab7844dc839109f66d2e72f52a",
        "domain": "omni",
        "question": "Let \\(ABC\\) be a triangle with \\(AB=13, BC=14\\), and \\(CA=15\\). Pick points \\(Q\\) and \\(R\\) on \\(AC\\) and \\(AB\\) such that \\(\\angle CBQ=\\angle BCR=90^{\\circ}\\). There exist two points \\(P_{1} \\neq P_{2}\\) in the plane of \\(ABC\\) such that \\(\\triangle P_{1}QR, \\triangle P_{2}QR\\), and \\(\\triangle ABC\\) are similar (with vertices in order). Compute the sum of the distances from \\(P_{1}\\) to \\(BC\\) and \\(P_{2}\\) to \\(BC\\).",
        "answer": "48",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_11",
        "source_index": 2318,
        "difficulty": 4.0
      },
      {
        "id": "ddc27d350c7973e063662b249c798ffb038d4c02f8529865d6926e2560f96134",
        "domain": "omni",
        "question": "A $5 \\times 5$ square grid has the number -3 written in the upper-left square and the number 3 written in the lower-right square. In how many ways can the remaining squares be filled in with integers so that any two adjacent numbers differ by 1, where two squares are adjacent if they share a common edge (but not if they share only a corner)?",
        "answer": "250",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 800,
        "difficulty": 4.5
      },
      {
        "id": "638ed7ed33c218183067da04ddeb7cb62021e9371b47226311c86caf911438cd",
        "domain": "omni",
        "question": "Let $S = \\{(x,y) | x = 1, 2, \\ldots, 1993, y = 1, 2, 3, 4\\}$. If $T \\subset S$ and there aren't any squares in $T.$ Find the maximum possible value of $|T|.$ The squares in T use points in S as vertices.",
        "answer": "5183",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 242,
        "difficulty": 8.0
      },
      {
        "id": "789536cb7eb9436cbc7a6afc3649791038adccdabd7e25a29f1939fcdeed651d",
        "domain": "omni",
        "question": "There is a grid of height 2 stretching infinitely in one direction. Between any two edge-adjacent cells of the grid, there is a door that is locked with probability $\\frac{1}{2}$ independent of all other doors. Philip starts in a corner of the grid (in the starred cell). Compute the expected number of cells that Philip can reach, assuming he can only travel between cells if the door between them is unlocked.",
        "answer": "\\frac{32}{7}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1181,
        "difficulty": 5.0
      },
      {
        "id": "a666f563bf2bcaae16def1cd4404576ba33a5fcbb53f28b59b7ea9ffa198e35a",
        "domain": "omni",
        "question": "Let $\\Omega$ be a circle of radius 8 centered at point $O$, and let $M$ be a point on $\\Omega$. Let $S$ be the set of points $P$ such that $P$ is contained within $\\Omega$, or such that there exists some rectangle $A B C D$ containing $P$ whose center is on $\\Omega$ with $A B=4, B C=5$, and $B C \\| O M$. Find the area of $S$.",
        "answer": "164+64 \\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2088,
        "difficulty": 4.0
      },
      {
        "id": "d0685d4009ee36686cf82a326601602d9e7588b6ae3c7be1683c0001e6d659a7",
        "domain": "omni",
        "question": "Let $AD, BE$, and $CF$ be segments sharing a common midpoint, with $AB < AE$ and $BC < BF$. Suppose that each pair of segments forms a $60^{\\circ}$ angle, and that $AD=7, BE=10$, and $CF=18$. Let $K$ denote the sum of the areas of the six triangles $\\triangle ABC, \\triangle BCD, \\triangle CDE, \\triangle DEF, \\triangle EFA$, and $\\triangle FAB$. Compute $K \\sqrt{3}$.",
        "answer": "141",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 848,
        "difficulty": 5.5
      },
      {
        "id": "8f44f54599fbc0faae64eb36723ab5ab54233fba4dec8bb2c9621ffca6583209",
        "domain": "omni",
        "question": "There are 12 students in a classroom; 6 of them are Democrats and 6 of them are Republicans. Every hour the students are randomly separated into four groups of three for political debates. If a group contains students from both parties, the minority in the group will change his/her political alignment to that of the majority at the end of the debate. What is the expected amount of time needed for all 12 students to have the same political alignment, in hours?",
        "answer": "\\frac{341}{55}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1879,
        "difficulty": 5.0
      },
      {
        "id": "27d95d023b687c86784fdb6cc06ed3af0935954e4c962012c2a503eb3835caef",
        "domain": "omni",
        "question": "Two random points are chosen on a segment and the segment is divided at each of these two points. Of the three segments obtained, find the probability that the largest segment is more than three times longer than the smallest segment.",
        "answer": "\\frac{27}{35}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1808,
        "difficulty": 5.25
      },
      {
        "id": "84f25e6a3c81076e46dce7f108917db34198c07094259daa0b1a8443a82545a9",
        "domain": "omni",
        "question": "Find the maximum number of points $X_{i}$ such that for each $i$, $\\triangle A B X_{i} \\cong \\triangle C D X_{i}$.",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "apmoapmo_sol",
        "source_index": 1543,
        "difficulty": 6.0
      },
      {
        "id": "2347afc58861d40e9d6945f07ed6aadb4592a2eacfa01aa73910ee81b76b5f8b",
        "domain": "omni",
        "question": "The area of the triangular region bounded by the $x$-axis, the $y$-axis and the line with equation $y=2x-6$ is one-quarter of the area of the triangular region bounded by the $x$-axis, the line with equation $y=2x-6$ and the line with equation $x=d$, where $d>0$. What is the value of $d$?",
        "answer": "9",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "cayley",
        "source_index": 2839,
        "difficulty": 2.0
      },
      {
        "id": "8a7f52c4bc7d30b9d76024167ebfe39ec61ec1c4a559788312e13bb927b74814",
        "domain": "omni",
        "question": "Triangle $ABC$ has side lengths $AB=19, BC=20$, and $CA=21$. Points $X$ and $Y$ are selected on sides $AB$ and $AC$, respectively, such that $AY=XY$ and $XY$ is tangent to the incircle of $\\triangle ABC$. If the length of segment $AX$ can be written as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers, compute $100 a+b$.",
        "answer": "6710",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 923,
        "difficulty": 5.5
      },
      {
        "id": "1e92a7808673ba645e508300483c456bb5a10faf81156605c6f73ca4254d2ba8",
        "domain": "omni",
        "question": "Aaron has 144 identical cubes, each with edge length 1 cm. He uses all of the cubes to construct a solid rectangular prism, which he places on a flat table. If the perimeter of the base of the prism is 20 cm, what is the sum of all possible heights of the prism?",
        "answer": "31",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "fermat",
        "source_index": 2852,
        "difficulty": 2.5
      },
      {
        "id": "7945a71a4473a0f2fb634dffe5fec718dcbe39f8f0140c52e03ee8e36fa54682",
        "domain": "omni",
        "question": "Ten distinct positive real numbers are given and the sum of each pair is written (So 45 sums). Between these sums there are 5 equal numbers. If we calculate product of each pair, find the biggest number $k$ such that there may be $k$ equal numbers between them.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "international_zhautykov_olympiad",
        "source_index": 3727,
        "difficulty": 7.0
      },
      {
        "id": "d83801c94b54820145a43ca50fac269be7e2b195e0e9ad61480f54ab45669f5c",
        "domain": "omni",
        "question": "How many different graphs with 9 vertices exist where each vertex is connected to 2 others?",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 1009,
        "difficulty": 5.0
      },
      {
        "id": "938cab3339f62e61e2eae2d6bf98a2000c8a4e2f4b027e7f09336e1a460fdf42",
        "domain": "omni",
        "question": "How many values of $x,-19<x<98$, satisfy $$\\cos ^{2} x+2 \\sin ^{2} x=1 ?$$",
        "answer": "38",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1453,
        "difficulty": 3.0
      },
      {
        "id": "b244a25eb18e2b66a3f074b01d88c16ca14fa06d0b22e9afa04c8d1ac04288fa",
        "domain": "omni",
        "question": "Let $S$ be a set of intervals defined recursively as follows: Initially, $[1,1000]$ is the only interval in $S$. If $l \\neq r$ and $[l, r] \\in S$, then both $\\left[l,\\left\\lfloor\\frac{l+r}{2}\\right\\rfloor\\right],\\left[\\left\\lfloor\\frac{l+r}{2}\\right\\rfloor+1, r\\right] \\in S$. An integer $i$ is chosen uniformly at random from the range $[1,1000]$. What is the expected number of intervals in $S$ which contain $i$?",
        "answer": "10.976",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1243,
        "difficulty": 6.0
      },
      {
        "id": "1b4a7fdde10d51131a01b3bc5ff2709337db4f0b67ba2f6a849fb740cf2acec6",
        "domain": "omni",
        "question": "Suppose that $P(x, y, z)$ is a homogeneous degree 4 polynomial in three variables such that $P(a, b, c)=P(b, c, a)$ and $P(a, a, b)=0$ for all real $a, b$, and $c$. If $P(1,2,3)=1$, compute $P(2,4,8)$.",
        "answer": "56",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1420,
        "difficulty": 5.0
      },
      {
        "id": "8104e4db204c3a256a9ac6d5dc90c13ccfa96856d3dd7cc088501c6748f618b1",
        "domain": "omni",
        "question": "A tightrope walker stands in the center of a rope of length 32 meters. Every minute she walks forward one meter with probability $3 / 4$ and backward one meter with probability $1 / 4$. What is the probability that she reaches the end in front of her before the end behind her?",
        "answer": "3^{16} / (3^{16}+1)",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 498,
        "difficulty": 5.25
      },
      {
        "id": "2019590ceb88635a3c307669eb7d90789579a880374e36a160b8a2ed52582c39",
        "domain": "omni",
        "question": "Knot is ready to face Gammadorf in a card game. In this game, there is a deck with twenty cards numbered from 1 to 20. Each player starts with a five card hand drawn from this deck. In each round, Gammadorf plays a card in his hand, then Knot plays a card in his hand. Whoever played a card with greater value gets a point. At the end of five rounds, the player with the most points wins. If Gammadorf starts with a hand of $1,5,10,15,20$, how many five-card hands of the fifteen remaining cards can Knot draw which always let Knot win (assuming he plays optimally)?",
        "answer": "2982",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 727,
        "difficulty": 4.5
      },
      {
        "id": "c0e9aa9d0883abddae8f0207abfa42248253641139b2cd6637bfb09c3f9fd9b2",
        "domain": "omni",
        "question": "Find the smallest positive real $\\alpha$, such that $$\\frac{x+y} {2}\\geq \\alpha\\sqrt{xy}+(1 - \\alpha)\\sqrt{\\frac{x^2+y^2}{2}}$$ for all positive reals $x, y$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Calculus -> Optimization -> Other",
        "source": "baltic_way",
        "source_index": 3887,
        "difficulty": 5.0
      },
      {
        "id": "29b0bfaeacc3b16476d245087bf6c474209604d7243559857791509451653ecf",
        "domain": "omni",
        "question": "Tetrahedron $A B C D$ has side lengths $A B=6, B D=6 \\sqrt{2}, B C=10, A C=8, C D=10$, and $A D=6$. The distance from vertex $A$ to face $B C D$ can be written as $\\frac{a \\sqrt{b}}{c}$, where $a, b, c$ are positive integers, $b$ is square-free, and $\\operatorname{gcd}(a, c)=1$. Find $100 a+10 b+c$.",
        "answer": "2851",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2573,
        "difficulty": 5.0
      },
      {
        "id": "af0a60547a8996b1090bda3dacb308feed455c5e45bd61f3c776ee9c9e8361c9",
        "domain": "omni",
        "question": "Let $x$ and $y$ be positive real numbers. Define $a=1+\\frac{x}{y}$ and $b=1+\\frac{y}{x}$. If $a^{2}+b^{2}=15$, compute $a^{3}+b^{3}$.",
        "answer": "50",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1609,
        "difficulty": 5.0
      },
      {
        "id": "75f9e72c6e0e6c234c8c2863a77576e6a860e854dc9edc504391202c4c19aed8",
        "domain": "omni",
        "question": "Augustin has six $1 \\times 2 \\times \\pi$ bricks. He stacks them, one on top of another, to form a tower six bricks high. Each brick can be in any orientation so long as it rests flat on top of the next brick below it (or on the floor). How many distinct heights of towers can he make?",
        "answer": "28",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 642,
        "difficulty": 4.0
      },
      {
        "id": "50bee68d78bc00933b87eabb5368acbf3149ff33b70184edb8f2fa4e5eb5fb20",
        "domain": "omni",
        "question": "Compute the number of functions $f:\\{1,2, \\ldots, 9\\} \\rightarrow\\{1,2, \\ldots, 9\\}$ which satisfy $f(f(f(f(f(x)))))=$ $x$ for each $x \\in\\{1,2, \\ldots, 9\\}$.",
        "answer": "3025",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_11",
        "source_index": 2586,
        "difficulty": 4.5
      },
      {
        "id": "e31fcd74f2e078e0956f2afcc5918c274a6678610bca02ab9d00a34d754ed797",
        "domain": "omni",
        "question": "For how many positive integers $n \\leq 100$ is it true that $10 n$ has exactly three times as many positive divisors as $n$ has?",
        "answer": "28",
        "topic": "Mathematics -> Number Theory -> Divisor Function -> Other; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1929,
        "difficulty": 4.5
      },
      {
        "id": "31d71c87343499a61e6e3baf285cc54d7a638fa37bb8a6ad24a102ed3c67b689",
        "domain": "omni",
        "question": "Let $n$ be a positive integer. Claudio has $n$ cards, each labeled with a different number from 1 to n. He takes a subset of these cards, and multiplies together the numbers on the cards. He remarks that, given any positive integer $m$, it is possible to select some subset of the cards so that the difference between their product and $m$ is divisible by 100. Compute the smallest possible value of $n$.",
        "answer": "17",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 964,
        "difficulty": 5.25
      },
      {
        "id": "c86bfaa185698423727c89aef981e3fdd030a2cefdcc0cd42556ffae34217798",
        "domain": "omni",
        "question": "Each cell of a $2 \\times 5$ grid of unit squares is to be colored white or black. Compute the number of such colorings for which no $2 \\times 2$ square is a single color.",
        "answer": "634",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1844,
        "difficulty": 5.0
      },
      {
        "id": "7ed55fe01ebda329f2b1c76db3bdcb55ea99521df689e2c0202b473145ae77e9",
        "domain": "omni",
        "question": "Find the number of ordered pairs of integers $(a, b) \\in\\{1,2, \\ldots, 35\\}^{2}$ (not necessarily distinct) such that $a x+b$ is a \"quadratic residue modulo $x^{2}+1$ and 35 \", i.e. there exists a polynomial $f(x)$ with integer coefficients such that either of the following equivalent conditions holds: - there exist polynomials $P, Q$ with integer coefficients such that $f(x)^{2}-(a x+b)=\\left(x^{2}+1\\right) P(x)+35 Q(x)$ - or more conceptually, the remainder when (the polynomial) $f(x)^{2}-(a x+b)$ is divided by (the polynomial) $x^{2}+1$ is a polynomial with (integer) coefficients all divisible by 35 .",
        "answer": "225",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory; Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1355,
        "difficulty": 5.5
      },
      {
        "id": "ccf59419b52537f8975a43d9c1d5569b8b1be0f5ea9b087a656ed35212adbcb3",
        "domain": "omni",
        "question": "16 students took part in a competition. All problems were multiple choice style. Each problem had four choices. It was said that any two students had at most one answer in common, find the maximum number of problems.",
        "answer": "5",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 145,
        "difficulty": 5.5
      },
      {
        "id": "2909af0e7cfbc1ad569baa7409c94755b612186a2f1be3d5cf3b10098e72a243",
        "domain": "omni",
        "question": "You are given an unlimited supply of red, blue, and yellow cards to form a hand. Each card has a point value and your score is the sum of the point values of those cards. The point values are as follows: the value of each red card is 1 , the value of each blue card is equal to twice the number of red cards, and the value of each yellow card is equal to three times the number of blue cards. What is the maximum score you can get with fifteen cards?",
        "answer": "168",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1283,
        "difficulty": 5.0
      },
      {
        "id": "d6b6dc77e3790767133973ebbee97df8b6ecc95dcaba15211e29c112dc5c6d93",
        "domain": "omni",
        "question": "Kevin writes down the positive integers $1,2, \\ldots, 15$ on a blackboard. Then, he repeatedly picks two random integers $a, b$ on the blackboard, erases them, and writes down $\\operatorname{gcd}(a, b)$ and $\\operatorname{lcm}(a, b)$. He does this until he is no longer able to change the set of numbers written on the board. Find the maximum sum of the numbers on the board after this process.",
        "answer": "360864",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 2554,
        "difficulty": 5.0
      },
      {
        "id": "2ca9799e25e2ec99cfbd739e080fb75405c365d4002c130a3da50ccf9593302c",
        "domain": "omni",
        "question": "The integer 119 is a multiple of which number?",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "cayley",
        "source_index": 3435,
        "difficulty": 1.5
      },
      {
        "id": "96436b508bf67308d43ef1ca170cce4d42b4c7957498aa103e044356d79b4c49",
        "domain": "omni",
        "question": "Let $x, y$, and $z$ be positive real numbers such that $(x \\cdot y)+z=(x+z) \\cdot(y+z)$. What is the maximum possible value of $x y z$?",
        "answer": "1/27",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1003,
        "difficulty": 5.0
      },
      {
        "id": "679146602cbb853a4d327e7c14fa4edee0f0918884e91eec7dddc3aa63b4c581",
        "domain": "omni",
        "question": "An infinite sequence $ \\,x_{0},x_{1},x_{2},\\ldots \\,$ of real numbers is said to be [b]bounded[/b] if there is a constant $ \\,C\\,$ such that $ \\, \\vert x_{i} \\vert \\leq C\\,$ for every $ \\,i\\geq 0$. Given any real number $ \\,a > 1,\\,$ construct a bounded infinite sequence $ x_{0},x_{1},x_{2},\\ldots \\,$ such that\n\\[ \\vert x_{i} \\minus{} x_{j} \\vert \\vert i \\minus{} j \\vert^{a}\\geq 1\n\\]\nfor every pair of distinct nonnegative integers $ i, j$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "imo",
        "source_index": 4407,
        "difficulty": 8.5
      },
      {
        "id": "09535a214cdaa17a15c1c24675b6e28c22b8fbd13737409c5ec68ca08d771667",
        "domain": "omni",
        "question": "How many integers are greater than $\\sqrt{15}$ and less than $\\sqrt{50}$?",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2986,
        "difficulty": 2.0
      },
      {
        "id": "df957c829f5a6b1f7915db801e91112e71664bb97ae0ca19a132753294b80815",
        "domain": "omni",
        "question": "Find the number of terms $n \\leq 2012$ such that $a_{n}=\\frac{3^{n+1}-1}{2}$ is divisible by 7.",
        "answer": "335",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2409,
        "difficulty": 5.0
      },
      {
        "id": "93648412e923f0704b63139fc1d4b9badea77b6f8048ba4020d3c345bba15e99",
        "domain": "omni",
        "question": "There are two buildings facing each other, each 5 stories high. How many ways can Kevin string ziplines between the buildings so that: (a) each zipline starts and ends in the middle of a floor. (b) ziplines can go up, stay flat, or go down, but can't touch each other (this includes touching at their endpoints). Note that you can't string a zipline between two floors of the same building.",
        "answer": "252",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1744,
        "difficulty": 4.0
      },
      {
        "id": "1f2d15e803cd1e60072244137736ddc29d9e7c261d27f3ec7dcdca3dab0ca396",
        "domain": "omni",
        "question": "A repunit is a positive integer, all of whose digits are 1s. Let $a_{1}<a_{2}<a_{3}<\\ldots$ be a list of all the positive integers that can be expressed as the sum of distinct repunits. Compute $a_{111}$.",
        "answer": "1223456",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2581,
        "difficulty": 5.0
      },
      {
        "id": "521932a6460e0aacf5738fd987a61a27fd34247cc4a2af28f20ac8942aa06acc",
        "domain": "omni",
        "question": "Quadrilateral $A B C D$ satisfies $A B=8, B C=5, C D=17, D A=10$. Let $E$ be the intersection of $A C$ and $B D$. Suppose $B E: E D=1: 2$. Find the area of $A B C D$.",
        "answer": "60",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2406,
        "difficulty": 4.0
      },
      {
        "id": "542123490fdf42c62bdeb0e2591da58a6fab49881c7afce3da27943892cb6fd7",
        "domain": "omni",
        "question": "Lucas writes two distinct positive integers on a whiteboard. He decreases the smaller number by 20 and increases the larger number by 23 , only to discover the product of the two original numbers is equal to the product of the two altered numbers. Compute the minimum possible sum of the original two numbers on the board.",
        "answer": "321",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1786,
        "difficulty": 5.0
      },
      {
        "id": "f237683825291b31b805fc94b028bee510441f587bef9dce5d3bd12c86c43fb7",
        "domain": "omni",
        "question": "A $3 \\times 3$ table starts with every entry equal to 0 and is modified using the following steps: (i) adding 1 to all three numbers in any row; (ii) adding 2 to all three numbers in any column. After step (i) has been used a total of $a$ times and step (ii) has been used a total of $b$ times, the table appears as \\begin{tabular}{|l|l|l|} \\hline 7 & 1 & 5 \\\\ \\hline 9 & 3 & 7 \\\\ \\hline 8 & 2 & 6 \\\\ \\hline \\end{tabular} shown. What is the value of $a+b$?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2885,
        "difficulty": 3.0
      },
      {
        "id": "107bb1109b2d4062c4763bc5e09b0ab3ae95b5a2653e2d936da29398b5dba165",
        "domain": "omni",
        "question": "Fisica and Ritmo discovered a piece of Notalium shaped like a rectangular box, and wanted to find its volume. To do so, Fisica measured its three dimensions using a ruler with infinite precision, multiplied the results and rounded the product to the nearest cubic centimeter, getting a result of 2017 cubic centimeters. Ritmo, on the other hand, measured each dimension to the nearest centimeter and multiplied the rounded measurements, getting a result of $V$ cubic centimeters. Find the positive difference between the least and greatest possible positive values for $V$.",
        "answer": "7174",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2607,
        "difficulty": 5.25
      },
      {
        "id": "82a844d6e662aa2fbb93ec0f1ef6187be7b2bcec1a98b8f21adb8d5935bebce5",
        "domain": "omni",
        "question": "A sequence has 101 terms, each of which is a positive integer. If a term, $n$, is even, the next term is equal to $\\frac{1}{2}n+1$. If a term, $n$, is odd, the next term is equal to $\\frac{1}{2}(n+1)$. If the first term is 16, what is the 101st term?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2888,
        "difficulty": 2.5
      },
      {
        "id": "956876d47c772fd738f3b58082e2ada4ca4eed534e16d97779b6c12f31cf3eeb",
        "domain": "omni",
        "question": "Let $a$ and $b$ be positive integers for which $45a+b=2021$. What is the minimum possible value of $a+b$?",
        "answer": "85",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3182,
        "difficulty": 2.0
      },
      {
        "id": "2515828c9f103b69816b9cb60800e219cd490e875b3895c5cd522311c76dd09a",
        "domain": "omni",
        "question": "In the future, MIT has attracted so many students that its buildings have become skyscrapers. Ben and Jerry decide to go ziplining together. Ben starts at the top of the Green Building, and ziplines to the bottom of the Stata Center. After waiting $a$ seconds, Jerry starts at the top of the Stata Center, and ziplines to the bottom of the Green Building. The Green Building is 160 meters tall, the Stata Center is 90 meters tall, and the two buildings are 120 meters apart. Furthermore, both zipline at 10 meters per second. Given that Ben and Jerry meet at the point where the two ziplines cross, compute $100 a$.",
        "answer": "740",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2330,
        "difficulty": 4.0
      },
      {
        "id": "d45b2ffbadc3bd0c608e278fe5792244bd9d5611c9cbbc6b2228efc755fd3024",
        "domain": "omni",
        "question": "You are the general of an army. You and the opposing general both have an equal number of troops to distribute among three battlefields. Whoever has more troops on a battlefield always wins (you win ties). An order is an ordered triple of non-negative real numbers $(x, y, z)$ such that $x+y+z=1$, and corresponds to sending a fraction $x$ of the troops to the first field, $y$ to the second, and $z$ to the third. Suppose that you give the order $\\left(\\frac{1}{4}, \\frac{1}{4}, \\frac{1}{2}\\right)$ and that the other general issues an order chosen uniformly at random from all possible orders. What is the probability that you win two out of the three battles?",
        "answer": "\\sqrt[5]{8}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 2357,
        "difficulty": 4.5
      },
      {
        "id": "d8c88ab42f895904820deea1a574516d8127bb92def994cc36ac9015dc4fc7bf",
        "domain": "omni",
        "question": "Suppose that $a$ and $b$ are integers with $4<a<b<22$. If the average (mean) of the numbers $4, a, b, 22$ is 13, how many possible pairs $(a, b)$ are there?",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2751,
        "difficulty": 2.5
      },
      {
        "id": "22173a486b258c4bdb9fcdbac72274cbba4f024b98b2dbc89edbdeb1a1a2cf73",
        "domain": "omni",
        "question": "Knowing that the system\n\\[x + y + z = 3,\\]\\[x^3 + y^3 + z^3 = 15,\\]\\[x^4 + y^4 + z^4 = 35,\\]\nhas a real solution $x, y, z$ for which $x^2 + y^2 + z^2 < 10$, find the value of $x^5 + y^5 + z^5$ for that solution.",
        "answer": "83",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "imo_longlists",
        "source_index": 4023,
        "difficulty": 5.5
      },
      {
        "id": "a8647937fcc3cb3e9f235915be8d18ac405079781808978ed7ea3968a75962d0",
        "domain": "omni",
        "question": "Find the smallest integer $n$ such that each subset of $\\{1,2,\\ldots, 2004\\}$ with $n$ elements has two distinct elements $a$ and $b$ for which $a^2-b^2$ is a multiple of $2004$.",
        "answer": "1003",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Prime Numbers",
        "source": "rioplatense_mathematical_olympiad_level",
        "source_index": 3870,
        "difficulty": 6.0
      },
      {
        "id": "05260b2c4f231f0fff375387cbb378330af2ff932271e22b15aae0ca0dc134c5",
        "domain": "omni",
        "question": "Compute the number of permutations $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$ so that for all (not necessarily distinct) $m, n \\in\\{1,2, \\ldots, 10\\}$ where $m+n$ is prime, $\\pi(m)+\\pi(n)$ is prime.",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 1309,
        "difficulty": 5.0
      },
      {
        "id": "c4db4b1dd3fcb90338da411dd17475cedfe63427453d5ffa1f1bccce58842ab1",
        "domain": "omni",
        "question": "What is the least number of gumballs that Wally must buy to guarantee that he receives 3 gumballs of the same colour?",
        "answer": "8",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 3205,
        "difficulty": 1.5
      },
      {
        "id": "1b327ad85eac08c1f81e4667b2f58f9aae43251f03a0e9363fefff3f77d0b9e6",
        "domain": "omni",
        "question": "The expression $\\left(1+\\frac{1}{2}\\right)\\left(1+\\frac{1}{3}\\right)\\left(1+\\frac{1}{4}\\right)\\left(1+\\frac{1}{5}\\right)\\left(1+\\frac{1}{6}\\right)\\left(1+\\frac{1}{7}\\right)\\left(1+\\frac{1}{8}\\right)\\left(1+\\frac{1}{9}\\right)$ is equal to what?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 3425,
        "difficulty": 2.5
      },
      {
        "id": "4685e163857b4ff0d30108710d65ff4680858b933f7fbe9d80865ac75be69ca0",
        "domain": "omni",
        "question": "The function $f(n)$ is defined on the positive integers and takes non-negative integer values. $f(2)=0,f(3)>0,f(9999)=3333$ and for all $m,n:$ \\[ f(m+n)-f(m)-f(n)=0 \\text{ or } 1. \\] Determine $f(1982)$.",
        "answer": "660",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "imo",
        "source_index": 3818,
        "difficulty": 7.5
      },
      {
        "id": "38d725e6b5bc3f5535f395c10ff2767148b35adea71812b86b33e82c84c1d523",
        "domain": "omni",
        "question": "Alain and Louise are driving on a circular track with radius 25 km. Alain leaves the starting line first, going clockwise at 80 km/h. Fifteen minutes later, Louise leaves the same starting line, going counterclockwise at 100 km/h. For how many hours will Louise have been driving when they pass each other for the fourth time?",
        "answer": "\\\\frac{10\\\\pi-1}{9}",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 3161,
        "difficulty": 2.0
      },
      {
        "id": "3d372f7fef43956989b038f678e29d5c4fadd3736f13d12166b0c5d24fdd77c3",
        "domain": "omni",
        "question": "Find the number of real zeros of $x^{3}-x^{2}-x+2$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Calculus -> Differential Calculus -> Derivatives",
        "source": "HMMT_2",
        "source_index": 385,
        "difficulty": 3.5
      },
      {
        "id": "bae5749b41050a1df2c19f47abdcdc465fa19cf1b61c0085e3983a5baf938b4d",
        "domain": "omni",
        "question": "Find the volume of the three-dimensional solid given by the inequality $\\sqrt{x^{2}+y^{2}}+$ $|z| \\leq 1$.",
        "answer": "2 \\pi / 3",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume",
        "source": "HMMT_2",
        "source_index": 811,
        "difficulty": 4.5
      },
      {
        "id": "be883d3628d16c129029c309e965755021661ca4077a0603b1a66506bd39ff5b",
        "domain": "omni",
        "question": "A basket contains 12 apples and 15 bananas. If 3 more bananas are added to the basket, what fraction of the fruit in the basket will be bananas?",
        "answer": "\\frac{3}{5}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3406,
        "difficulty": 1.5
      },
      {
        "id": "535530d06acfc95a15ef75e1e2bbb9255f6201e440c2152e647b94f47057c418",
        "domain": "omni",
        "question": "An $n$-string is a string of digits formed by writing the numbers $1,2, \\ldots, n$ in some order (in base ten). For example, one possible 10-string is $$35728910461$$ What is the smallest $n>1$ such that there exists a palindromic $n$-string?",
        "answer": "19",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 713,
        "difficulty": 4.5
      },
      {
        "id": "4f84a6c9e22e94bc2ab5ae9fe50dd6e15160cd2d805fc0ba6aa54b1ef8c1a2a9",
        "domain": "omni",
        "question": "In a $16 \\times 16$ table of integers, each row and column contains at most 4 distinct integers. What is the maximum number of distinct integers that there can be in the whole table?",
        "answer": "49",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1397,
        "difficulty": 4.0
      },
      {
        "id": "95ea8627ad7396517a5643cbd0b6ad4f01dbfaa8221b97c4660308c5e16cc373",
        "domain": "omni",
        "question": "We colour all the sides and diagonals of a regular polygon $P$ with $43$ vertices either\nred or blue in such a way that every vertex is an endpoint of $20$ red segments and $22$ blue segments.\nA triangle formed by vertices of $P$ is called monochromatic if all of its sides have the same colour.\nSuppose that there are $2022$ blue monochromatic triangles. How many red monochromatic triangles\nare there?",
        "answer": "859",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imc",
        "source_index": 3664,
        "difficulty": 8.0
      },
      {
        "id": "0faa0b0b845ba72741d532991854fc74adf250aa9121d977051bae7e063f30ed",
        "domain": "omni",
        "question": "Let $S$ be the sum of all the real coefficients of the expansion of $(1+i x)^{2009}$. What is $\\log _{2}(S)$ ?",
        "answer": "1004",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1234,
        "difficulty": 4.0
      },
      {
        "id": "64b12006119459049296e4c8f0fe960a13ee14bfde20817139e9ac629def8ff0",
        "domain": "omni",
        "question": "Four points are chosen uniformly and independently at random in the interior of a given circle. Find the probability that they are the vertices of a convex quadrilateral.",
        "answer": "1 - \\frac{35}{12 \\pi^2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "putnam",
        "source_index": 3186,
        "difficulty": 8.0
      },
      {
        "id": "332a09892fc986f8e0dd238754b6f5138398202871374f429acbbc9468175506",
        "domain": "omni",
        "question": "A set consists of five different odd positive integers, each greater than 2. When these five integers are multiplied together, their product is a five-digit integer of the form $AB0AB$, where $A$ and $B$ are digits with $A \\neq 0$ and $A \\neq B$. (The hundreds digit of the product is zero.) For example, the integers in the set $\\{3,5,7,13,33\\}$ have a product of 45045. In total, how many different sets of five different odd positive integers have these properties?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2744,
        "difficulty": 2.5
      },
      {
        "id": "41be6c73081ec989f7bb49366e9b8c1c2f418a5fb4708aa4861b8d92e495fffd",
        "domain": "omni",
        "question": "Let $ABC$ be a right triangle with $\\angle A=90^{\\circ}$. Let $D$ be the midpoint of $AB$ and let $E$ be a point on segment $AC$ such that $AD=AE$. Let $BE$ meet $CD$ at $F$. If $\\angle BFC=135^{\\circ}$, determine $BC/AB$.",
        "answer": "\\frac{\\sqrt{13}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 579,
        "difficulty": 5.0
      },
      {
        "id": "a98dfceb0b5b977c9f55d9f4378f6dc44ab6131b8607ccb1d02da03c381c5dc1",
        "domain": "omni",
        "question": "Find the number of 7 -tuples $\\left(n_{1}, \\ldots, n_{7}\\right)$ of integers such that $$\\sum_{i=1}^{7} n_{i}^{6}=96957$$",
        "answer": "2688",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1520,
        "difficulty": 5.0
      },
      {
        "id": "22f34684f18d21ddf1e3fa6b7969daf260b33fba3fcac8a5cad20c07b6a54484",
        "domain": "omni",
        "question": "Sherry is waiting for a train. Every minute, there is a $75 \\%$ chance that a train will arrive. However, she is engrossed in her game of sudoku, so even if a train arrives she has a $75 \\%$ chance of not noticing it (and hence missing the train). What is the probability that Sherry catches the train in the next five minutes?",
        "answer": "1-\\left(\\frac{13}{16}\\right)^{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1099,
        "difficulty": 4.5
      },
      {
        "id": "006a6aa7c4af32ea4c034e861a8efa07ae86d04bba87e5b8e0c9a17fe4625a8e",
        "domain": "omni",
        "question": "The numbers $1,2, \\ldots, 10$ are written in a circle. There are four people, and each person randomly selects five consecutive integers (e.g. $1,2,3,4,5$, or $8,9,10,1,2$). If the probability that there exists some number that was not selected by any of the four people is $p$, compute $10000p$.",
        "answer": "3690",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2603,
        "difficulty": 5.0
      },
      {
        "id": "31d4abb8642770a31298f9102e61beb8e3d40e691af097fc9c9e668b70f69ffe",
        "domain": "omni",
        "question": "Let $p, q, r$ be primes such that $2 p+3 q=6 r$. Find $p+q+r$.",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2496,
        "difficulty": 4.0
      },
      {
        "id": "aebd29b82a1c1e387872445471c59b3f5fd76985c4f441da2c2ae66f3e21fdf2",
        "domain": "omni",
        "question": "A 24-hour digital clock shows times $h: m: s$, where $h, m$, and $s$ are integers with $0 \\leq h \\leq 23$, $0 \\leq m \\leq 59$, and $0 \\leq s \\leq 59$. How many times $h: m: s$ satisfy $h+m=s$?",
        "answer": "1164",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2289,
        "difficulty": 4.0
      },
      {
        "id": "2056a3ffcabbf92e98b983e862971cc184a45bbf90fd48cd8fd2d2731143eda0",
        "domain": "omni",
        "question": "In triangle $A B C$, let the parabola with focus $A$ and directrix $B C$ intersect sides $A B$ and $A C$ at $A_{1}$ and $A_{2}$, respectively. Similarly, let the parabola with focus $B$ and directrix $C A$ intersect sides $B C$ and $B A$ at $B_{1}$ and $B_{2}$, respectively. Finally, let the parabola with focus $C$ and directrix $A B$ intersect sides $C A$ and $C B$ at $C_{1}$ and $C_{2}$, respectively. If triangle $A B C$ has sides of length 5,12, and 13, find the area of the triangle determined by lines $A_{1} C_{2}, B_{1} A_{2}$ and $C_{1} B_{2}$.",
        "answer": "\\frac{6728}{3375}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1734,
        "difficulty": 5.0
      },
      {
        "id": "8b9ad4916720b8851e75f5d273e405c4f1361d6337d16973950d9626050c01a5",
        "domain": "omni",
        "question": "Let $A B C D$ be a parallelogram with $A B=8, A D=11$, and $\\angle B A D=60^{\\circ}$. Let $X$ be on segment $C D$ with $C X / X D=1 / 3$ and $Y$ be on segment $A D$ with $A Y / Y D=1 / 2$. Let $Z$ be on segment $A B$ such that $A X, B Y$, and $D Z$ are concurrent. Determine the area of triangle $X Y Z$.",
        "answer": "\\frac{19 \\sqrt{3}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 598,
        "difficulty": 5.0
      },
      {
        "id": "943f12c7a25396090f5e1b5d18c48965628db39ec86c2f238e460b5ed68c25cf",
        "domain": "omni",
        "question": "Carl is on a vertex of a regular pentagon. Every minute, he randomly selects an adjacent vertex (each with probability $\\frac{1}{2}$ ) and walks along the edge to it. What is the probability that after 10 minutes, he ends up where he had started?",
        "answer": "\\frac{127}{512}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1938,
        "difficulty": 4.5
      },
      {
        "id": "d14fb75b4558ff5fa3ff54c68e8546652420a643e7979cf6447f185c6e01a720",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that the $73$ fractions $\\frac{19}{n+21}, \\frac{20}{n+22},\\frac{21}{n+23},...,\\frac{91}{n+93}$ are all irreducible.",
        "answer": "95",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "cono_sur_olympiad",
        "source_index": 3639,
        "difficulty": 5.0
      },
      {
        "id": "b190f6c16b7c338c617484dd735acfc0cdbe5032ca95115b8093a17d5b0adad4",
        "domain": "omni",
        "question": "Find the probability that a monkey typing randomly on a typewriter will type the string 'abc' before 'aaa'.",
        "answer": "\\frac{3}{7}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2401,
        "difficulty": 5.0
      },
      {
        "id": "bcf537073f3cf214bcdcc42ab086a88603b10cde87c317c923f08f1fc7a38e0a",
        "domain": "omni",
        "question": "Let $\\omega_{1}, \\omega_{2}, \\ldots, \\omega_{100}$ be the roots of $\\frac{x^{101}-1}{x-1}$ (in some order). Consider the set $S=\\left\\{\\omega_{1}^{1}, \\omega_{2}^{2}, \\omega_{3}^{3}, \\ldots, \\omega_{100}^{100}\\right\\}$. Let $M$ be the maximum possible number of unique values in $S$, and let $N$ be the minimum possible number of unique values in $S$. Find $M-N$.",
        "answer": "98",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1571,
        "difficulty": 6.0
      },
      {
        "id": "745f4feeb0efd455753ffe2ffdf4c3550b375011a0034c40f7c23c5a11b25256",
        "domain": "omni",
        "question": "A perfect power is an integer $n$ that can be represented as $a^{k}$ for some positive integers $a \\geq 1$ and $k \\geq 2$. Find the sum of all prime numbers $0<p<50$ such that $p$ is 1 less than a perfect power.",
        "answer": "41",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2660,
        "difficulty": 5.0
      },
      {
        "id": "ab22c44d583936fac2591423b12bdbf62064c05af5b382b735d16675e60bd6c8",
        "domain": "omni",
        "question": "Calculate the probability of the Alphas winning given the probability of the Reals hitting 0, 1, 2, 3, or 4 singles.",
        "answer": "\\frac{224}{243}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 948,
        "difficulty": 4.0
      },
      {
        "id": "66791a2725fbc6b708ea9db6dd1eaf9eeec3f1e6d5dfa14e140e17bb86eddb9c",
        "domain": "omni",
        "question": "Find the number of pentominoes (5-square polyominoes) that span a 3-by-3 rectangle, where polyominoes that are flips or rotations of each other are considered the same polyomino.",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 839,
        "difficulty": 3.5
      },
      {
        "id": "ca2fc1076576a645af6879cc5d8dd839c8f9369f7ea1c5f15ecc64bc2194bb44",
        "domain": "omni",
        "question": "Define a number to be an anti-palindrome if, when written in base 3 as $a_{n} a_{n-1} \\ldots a_{0}$, then $a_{i}+a_{n-i}=2$ for any $0 \\leq i \\leq n$. Find the number of anti-palindromes less than $3^{12}$ such that no two consecutive digits in base 3 are equal.",
        "answer": "126",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1850,
        "difficulty": 5.25
      },
      {
        "id": "8450f45d6c6120f6102de7a5385ad62b465b2a39909192ef8cb770166a587ba5",
        "domain": "omni",
        "question": "Complex number $\\omega$ satisfies $\\omega^{5}=2$. Find the sum of all possible values of $\\omega^{4}+\\omega^{3}+\\omega^{2}+\\omega+1$.",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "HMMT_11",
        "source_index": 2010,
        "difficulty": 4.0
      },
      {
        "id": "83ce287f9613ea253fbe401513271f0119a299a879d439b5996099472dfbec86",
        "domain": "omni",
        "question": "A sequence of real numbers $a_{0}, a_{1}, \\ldots, a_{9}$ with $a_{0}=0, a_{1}=1$, and $a_{2}>0$ satisfies $$a_{n+2} a_{n} a_{n-1}=a_{n+2}+a_{n}+a_{n-1}$$ for all $1 \\leq n \\leq 7$, but cannot be extended to $a_{10}$. In other words, no values of $a_{10} \\in \\mathbb{R}$ satisfy $$a_{10} a_{8} a_{7}=a_{10}+a_{8}+a_{7}$$ Compute the smallest possible value of $a_{2}$.",
        "answer": "\\sqrt{2}-1",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1900,
        "difficulty": 5.0
      },
      {
        "id": "fd22d34b3bcd8c1beec717b528cfab88d7719c61738dd4fd59d1238da279e15a",
        "domain": "omni",
        "question": "Consider an $8 \\times 8$ grid of squares. A rook is placed in the lower left corner, and every minute it moves to a square in the same row or column with equal probability (the rook must move; i.e. it cannot stay in the same square). What is the expected number of minutes until the rook reaches the upper right corner?",
        "answer": "70",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2038,
        "difficulty": 4.0
      },
      {
        "id": "bed2dfe88c3b32969ba0b17b05380d3d8d764d5b7c69972b0db8f71c7a4cb4d7",
        "domain": "omni",
        "question": "How many sequences of 0s and 1s are there of length 10 such that there are no three 0s or 1s consecutively anywhere in the sequence?",
        "answer": "178",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 776,
        "difficulty": 4.0
      },
      {
        "id": "182d0f29fb3f81552ae8cbe26f76fe6cffd8fc013af9a4bbbdd93554890cbe5a",
        "domain": "omni",
        "question": "Rosencrantz plays $n \\leq 2015$ games of question, and ends up with a win rate (i.e. $\\frac{\\# \\text { of games won }}{\\# \\text { of games played }}$ ) of $k$. Guildenstern has also played several games, and has a win rate less than $k$. He realizes that if, after playing some more games, his win rate becomes higher than $k$, then there must have been some point in time when Rosencrantz and Guildenstern had the exact same win-rate. Find the product of all possible values of $k$.",
        "answer": "\\frac{1}{2015}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2136,
        "difficulty": 4.0
      },
      {
        "id": "d3fc65cd6dfd5ddfad4b211c35e2bc0e722c91042b41c0024a22cf543bb981a3",
        "domain": "omni",
        "question": "Let $S=\\{1,2, \\ldots 2016\\}$, and let $f$ be a randomly chosen bijection from $S$ to itself. Let $n$ be the smallest positive integer such that $f^{(n)}(1)=1$, where $f^{(i)}(x)=f\\left(f^{(i-1)}(x)\\right)$. What is the expected value of $n$?",
        "answer": "\\frac{2017}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2424,
        "difficulty": 4.0
      },
      {
        "id": "6902ac8cae33b98cb9d5f02f823928683324b18486eb7dff50fc6ea31fc8bc96",
        "domain": "omni",
        "question": "Triangle $\\triangle P Q R$, with $P Q=P R=5$ and $Q R=6$, is inscribed in circle $\\omega$. Compute the radius of the circle with center on $\\overline{Q R}$ which is tangent to both $\\omega$ and $\\overline{P Q}$.",
        "answer": "\\frac{20}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 1871,
        "difficulty": 5.0
      },
      {
        "id": "67f30237e4dfa896f88d15a58fffadc598862b3e83a5a73a73c91073cd84f33f",
        "domain": "omni",
        "question": "Seven little children sit in a circle. The teacher distributes pieces of candy to the children in such a way that the following conditions hold. - Every little child gets at least one piece of candy. - No two little children have the same number of pieces of candy. - The numbers of candy pieces given to any two adjacent little children have a common factor other than 1. - There is no prime dividing every little child's number of candy pieces. What is the smallest number of pieces of candy that the teacher must have ready for the little children?",
        "answer": "44",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1794,
        "difficulty": 4.5
      },
      {
        "id": "079fb9bcaf94bd929efa5f2e311ec3485c1ad4c83742e0b3d02a29fa85dfa650",
        "domain": "omni",
        "question": "An up-right path between two lattice points $P$ and $Q$ is a path from $P$ to $Q$ that takes steps of length 1 unit either up or to the right. How many up-right paths from $(0,0)$ to $(7,7)$, when drawn in the plane with the line $y=x-2.021$, enclose exactly one bounded region below that line?",
        "answer": "637",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_2",
        "source_index": 1297,
        "difficulty": 4.0
      },
      {
        "id": "48d2c1a07d6bf83ded696e9db88417cb6c60339be40e40ead2a12c21235777d1",
        "domain": "omni",
        "question": "Let $z_{1}, z_{2}, z_{3}, z_{4}$ be the solutions to the equation $x^{4}+3 x^{3}+3 x^{2}+3 x+1=0$. Then $\\left|z_{1}\\right|+\\left|z_{2}\\right|+\\left|z_{3}\\right|+\\left|z_{4}\\right|$ can be written as $\\frac{a+b \\sqrt{c}}{d}$, where $c$ is a square-free positive integer, and $a, b, d$ are positive integers with $\\operatorname{gcd}(a, b, d)=1$. Compute $1000 a+100 b+10 c+d$.",
        "answer": "7152",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1421,
        "difficulty": 5.0
      },
      {
        "id": "e801e6209257656c1df26a2cfe95c4947e99a199fb9c87d41d354901bb9966f5",
        "domain": "omni",
        "question": "Call an positive integer almost-square if it can be written as $a \\cdot b$, where $a$ and $b$ are integers and $a \\leq b \\leq \\frac{4}{3} a$. How many almost-square positive integers are less than or equal to 1000000 ? Your score will be equal to $25-65 \\frac{|A-C|}{\\min (A, C)}$.",
        "answer": "130348",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 2500,
        "difficulty": 5.25
      },
      {
        "id": "12be0ce43d4a030b277523c4e07a3203cd577ca7abe33dceda7941d17b5758c8",
        "domain": "omni",
        "question": "For any finite set $S$, let $f(S)$ be the sum of the elements of $S$ (if $S$ is empty then $f(S)=0$). Find the sum over all subsets $E$ of $S$ of $\\frac{f(E)}{f(S)}$ for $S=\\{1,2, \\ldots, 1999\\}$.",
        "answer": "2^{1998}",
        "topic": "Mathematics -> Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1157,
        "difficulty": 5.0
      },
      {
        "id": "f052d9e70b5ee1436cd96a62cd51fd4eccaac0f304b173387c79404cf62bdc7c",
        "domain": "omni",
        "question": "Let $f(x)=x^{2}+6 x+7$. Determine the smallest possible value of $f(f(f(f(x))))$ over all real numbers $x$.",
        "answer": "23",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "HMMT_11",
        "source_index": 2081,
        "difficulty": 5.0
      },
      {
        "id": "b27fb1b4afe80dadb7e31aed3cb0eed950f25e2ac28cb83d79a248600f97fc61",
        "domain": "omni",
        "question": "The numbers $1,2, \\ldots, 10$ are randomly arranged in a circle. Let $p$ be the probability that for every positive integer $k<10$, there exists an integer $k^{\\prime}>k$ such that there is at most one number between $k$ and $k^{\\prime}$ in the circle. If $p$ can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
        "answer": "1390",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1755,
        "difficulty": 4.0
      },
      {
        "id": "61a4bebbc753a897ff4e6f86eaa8fa71186918714e4aa7f873b9c2134d903579",
        "domain": "omni",
        "question": "Find the number of positive integer divisors of 12 ! that leave a remainder of 1 when divided by 3.",
        "answer": "66",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2361,
        "difficulty": 4.0
      },
      {
        "id": "6f91b175e8d85813dee85f8705749d8822053a647d8493760d24a2d84a6b6cb2",
        "domain": "omni",
        "question": "In a certain country, there are 100 senators, each of whom has 4 aides. These senators and aides serve on various committees. A committee may consist either of 5 senators, of 4 senators and 4 aides, or of 2 senators and 12 aides. Every senator serves on 5 committees, and every aide serves on 3 committees. How many committees are there altogether?",
        "answer": "160",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1440,
        "difficulty": 4.0
      },
      {
        "id": "50ab9b23f9c372fa757d437b50e22aeb9a2d1625a962eb190c41aadaa6c4a5c1",
        "domain": "omni",
        "question": "If \\( 3x + 4 = x + 2 \\), what is the value of \\( x \\)?",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3094,
        "difficulty": 1.0
      },
      {
        "id": "859c6346b79a288965255792b9c34f9d6851ead7cda9b26410c0906093b36b51",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=A C=5$ and $B C=6$. Denote by $\\omega$ the circumcircle of $A B C$. We draw a circle $\\Omega$ which is externally tangent to $\\omega$ as well as to the lines $A B$ and $A C$ (such a circle is called an $A$-mixtilinear excircle). Find the radius of $\\Omega$.",
        "answer": "\\frac{75}{8}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1716,
        "difficulty": 4.5
      },
      {
        "id": "288df7dcb3f3000cef4619f25aacfca4a8d1da858df195547586db02b2bf4817",
        "domain": "omni",
        "question": "Let $r = \\sqrt{\\frac{\\sqrt{53}}{2} + \\frac{3}{2}}$. There is a unique triple of positive integers $(a, b, c)$ such that $r^{100} = 2r^{98} + 14r^{96} + 11r^{94} - r^{50} + ar^{46} + br^{44} + cr^{40}$. What is the value of $a^{2} + b^{2} + c^{2}$?",
        "answer": "15339",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2791,
        "difficulty": 2.5
      },
      {
        "id": "0467816f3a62c5063d3d120ba74fc0aa5d1aeb4cdfc63b263fcdee66b4c766e0",
        "domain": "omni",
        "question": "You have 128 teams in a single elimination tournament. The Engineers and the Crimson are two of these teams. Each of the 128 teams in the tournament is equally strong, so during each match, each team has an equal probability of winning. Now, the 128 teams are randomly put into the bracket. What is the probability that the Engineers play the Crimson sometime during the tournament?",
        "answer": "\\frac{1}{64}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 520,
        "difficulty": 3.5
      },
      {
        "id": "a30427c195e6b262163fe82ace9ba82f1d0e0650a32ec1d352cd460a5640e605",
        "domain": "omni",
        "question": "In each cell of a $4 \\times 4$ grid, one of the two diagonals is drawn uniformly at random. Compute the probability that the resulting 32 triangular regions can be colored red and blue so that any two regions sharing an edge have different colors.",
        "answer": "\\frac{1}{512}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1152,
        "difficulty": 5.5
      },
      {
        "id": "14ea471f7a04fb4482e5e43e70471e31473b946d2ce1b37559aa9614cc6c1053",
        "domain": "omni",
        "question": "Consider an equilateral triangle and a square both inscribed in a unit circle such that one side of the square is parallel to one side of the triangle. Compute the area of the convex heptagon formed by the vertices of both the triangle and the square.",
        "answer": "\\frac{3+\\sqrt{3}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1764,
        "difficulty": 5.0
      },
      {
        "id": "4297d9c2896199386786f130f69d2b446b7c1eb185e3d48c8695a08614018b0b",
        "domain": "omni",
        "question": "A restricted path of length $n$ is a path of length $n$ such that for all $i$ between 1 and $n-2$ inclusive, if the $i$th step is upward, the $i+1$st step must be rightward. Find the number of restricted paths that start at $(0,0)$ and end at $(7,3)$.",
        "answer": "56",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 909,
        "difficulty": 5.0
      },
      {
        "id": "5985b87694f5ad233eba27863d1d3abdf28bed4283115a787c9dba819e81197e",
        "domain": "omni",
        "question": "Define $\\phi^{!}(n)$ as the product of all positive integers less than or equal to $n$ and relatively prime to $n$. Compute the remainder when $$ \\sum_{\\substack{2 \\leq n \\leq 50 \\\\ \\operatorname{gcd}(n, 50)=1}} \\phi^{!}(n) $$ is divided by 50 .",
        "answer": "12",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 1031,
        "difficulty": 5.0
      },
      {
        "id": "3bc738c688e27db4e044b2f556260bd8db1b8cf7eddaaf91a52b9222700bdc68",
        "domain": "omni",
        "question": "Let $N$ be the number of triples of positive integers $(a, b, c)$ satisfying $a \\leq b \\leq c, \\quad \\operatorname{gcd}(a, b, c)=1, \\quad a b c=6^{2020}$. Compute the remainder when $N$ is divided by 1000.",
        "answer": "602",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1308,
        "difficulty": 4.0
      },
      {
        "id": "deeaa1575abfd6c6c878235445a2d38ee845466bc82c33958c940617b67176b8",
        "domain": "omni",
        "question": "Evaluate $\\sum_{n=2}^{17} \\frac{n^{2}+n+1}{n^{4}+2 n^{3}-n^{2}-2 n}$.",
        "answer": "\\frac{592}{969}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 953,
        "difficulty": 5.0
      },
      {
        "id": "5ab9f3946fc68ee7bcf19e561aefeeb2f4217ee184d566ef17ed237a373745f2",
        "domain": "omni",
        "question": "We have a polyhedron such that an ant can walk from one vertex to another, traveling only along edges, and traversing every edge exactly once. What is the smallest possible total number of vertices, edges, and faces of this polyhedron?",
        "answer": "20",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1140,
        "difficulty": 4.0
      },
      {
        "id": "9b5abb638578fe4f25dde659db33eea81f975d549100aed19b62bee11b7c3876",
        "domain": "omni",
        "question": "When $x=3$ and $y=4$, what is the value of the expression $xy-x$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3072,
        "difficulty": 1.5
      },
      {
        "id": "451b0938699b9ff1a371b5008081950a3e7fc5a5b4fe03f2adce7fe824cf4c62",
        "domain": "omni",
        "question": "If \\( x \\) and \\( y \\) are positive integers with \\( x>y \\) and \\( x+x y=391 \\), what is the value of \\( x+y \\)?",
        "answer": "39",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2699,
        "difficulty": 2.0
      },
      {
        "id": "e55c401c0ba993e8a25ece98971d65f9b09dc3335892af12810c127797465e46",
        "domain": "omni",
        "question": "On each cell of a $200 \\times 200$ grid, we place a car, which faces in one of the four cardinal directions. In a move, one chooses a car that does not have a car immediately in front of it, and slides it one cell forward. If a move would cause a car to exit the grid, the car is removed instead. The cars are placed so that there exists a sequence of moves that eventually removes all the cars from the grid. Across all such starting configurations, determine the maximum possible number of moves to do so.",
        "answer": "6014950",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1196,
        "difficulty": 7.0
      },
      {
        "id": "96c2d980269c391eee302f172390673434e810ac5936fa1ea37bd14f3eb40e75",
        "domain": "omni",
        "question": "Compute the number of integers \\(n \\in\\{1,2, \\ldots, 300\\}\\) such that \\(n\\) is the product of two distinct primes, and is also the length of the longest leg of some nondegenerate right triangle with integer side lengths.",
        "answer": "13",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1542,
        "difficulty": 5.25
      },
      {
        "id": "b5c66c1539ba781e2dfe72ffb13a35848eb40c06e7b6d31e78047672df1d6121",
        "domain": "omni",
        "question": "What is the value of $x$ if the three numbers $2, x$, and 10 have an average of $x$?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2667,
        "difficulty": 1.5
      },
      {
        "id": "c23558e89cfb5fe1b99c8747607e801b9fb18c20d3d0173f5416ace553149765",
        "domain": "omni",
        "question": "For dessert, Melinda eats a spherical scoop of ice cream with diameter 2 inches. She prefers to eat her ice cream in cube-like shapes, however. She has a special machine which, given a sphere placed in space, cuts it through the planes $x=n, y=n$, and $z=n$ for every integer $n$ (not necessarily positive). Melinda centers the scoop of ice cream uniformly at random inside the cube $0 \\leq x, y, z \\leq 1$, and then cuts it into pieces using her machine. What is the expected number of pieces she cuts the ice cream into?",
        "answer": "7+\\frac{13 \\pi}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2141,
        "difficulty": 5.0
      },
      {
        "id": "ff4852925c116ff06fed37418f32f41491bf3162b3003760204a05827b734583",
        "domain": "omni",
        "question": "In how many different places in the $xy$-plane can a third point, $R$, be placed so that $PQ = QR = PR$ if points $P$ and $Q$ are two distinct points in the $xy$-plane?",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "fermat",
        "source_index": 2806,
        "difficulty": 2.5
      },
      {
        "id": "c58700aac087fc9fdd70eb1cf23aa16838eefa4ebb69d79451325a88cd550423",
        "domain": "omni",
        "question": "Let $a, b, c, d, e, f$ be integers selected from the set $\\{1,2, \\ldots, 100\\}$, uniformly and at random with replacement. Set $M=a+2 b+4 c+8 d+16 e+32 f$. What is the expected value of the remainder when $M$ is divided by 64?",
        "answer": "\\frac{63}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1176,
        "difficulty": 4.0
      },
      {
        "id": "af6788bbaf262b5133f6cdf64ed2a10eb937ec2c9ab070b85c8e2a06be66af94",
        "domain": "omni",
        "question": "Convex quadrilateral $A B C D$ has right angles $\\angle A$ and $\\angle C$ and is such that $A B=B C$ and $A D=C D$. The diagonals $A C$ and $B D$ intersect at point $M$. Points $P$ and $Q$ lie on the circumcircle of triangle $A M B$ and segment $C D$, respectively, such that points $P, M$, and $Q$ are collinear. Suppose that $m \\angle A B C=160^{\\circ}$ and $m \\angle Q M C=40^{\\circ}$. Find $M P \\cdot M Q$, given that $M C=6$.",
        "answer": "36",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1514,
        "difficulty": 5.25
      },
      {
        "id": "00fe5ecadd779fbd79db4991b2a126eb4feb8d8a41262f83247d4817b8d5b550",
        "domain": "omni",
        "question": "The unknown real numbers $x, y, z$ satisfy the equations $$\\frac{x+y}{1+z}=\\frac{1-z+z^{2}}{x^{2}-x y+y^{2}} ; \\quad \\frac{x-y}{3-z}=\\frac{9+3 z+z^{2}}{x^{2}+x y+y^{2}}$$ Find $x$.",
        "answer": "\\sqrt[3]{14}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Cubic Functions -> Other",
        "source": "HMMT_2",
        "source_index": 802,
        "difficulty": 5.0
      },
      {
        "id": "5c675960ec93f3db6a9c49be5bc44f0ff45edb14636f18675b999cd6209fffb0",
        "domain": "omni",
        "question": "At Barker High School, a total of 36 students are on either the baseball team, the hockey team, or both. If there are 25 students on the baseball team and 19 students on the hockey team, how many students play both sports?",
        "answer": "8",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2739,
        "difficulty": 2.0
      },
      {
        "id": "db26cfdb410d03e322fa79c638467d012e6d4656c69cff3e15c384690d569a82",
        "domain": "omni",
        "question": "A triple of positive integers $(a, b, c)$ is tasty if $\\operatorname{lcm}(a, b, c) \\mid a+b+c-1$ and $a<b<c$. Find the sum of $a+b+c$ across all tasty triples.",
        "answer": "44",
        "topic": "Mathematics -> Number Theory -> Least Common Multiples (LCM)",
        "source": "HMMT_11",
        "source_index": 2028,
        "difficulty": 5.0
      },
      {
        "id": "11a720bac8b1671824060c9c7f511a418e11c36582c012e7f4c399fd7bb9d569",
        "domain": "omni",
        "question": "Caroline starts with the number 1, and every second she flips a fair coin; if it lands heads, she adds 1 to her number, and if it lands tails she multiplies her number by 2. Compute the expected number of seconds it takes for her number to become a multiple of 2021.",
        "answer": "4040",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_2",
        "source_index": 922,
        "difficulty": 5.25
      },
      {
        "id": "26bf9cadb30946f16495d9010d394393703d4dd13fa3b64464fbe08e677f573f",
        "domain": "omni",
        "question": "Divide an $m$-by-$n$ rectangle into $m n$ nonoverlapping 1-by-1 squares. A polyomino of this rectangle is a subset of these unit squares such that for any two unit squares $S, T$ in the polyomino, either (1) $S$ and $T$ share an edge or (2) there exists a positive integer $n$ such that the polyomino contains unit squares $S_{1}, S_{2}, S_{3}, \\ldots, S_{n}$ such that $S$ and $S_{1}$ share an edge, $S_{n}$ and $T$ share an edge, and for all positive integers $k<n, S_{k}$ and $S_{k+1}$ share an edge. We say a polyomino of a given rectangle spans the rectangle if for each of the four edges of the rectangle the polyomino contains a square whose edge lies on it. What is the minimum number of unit squares a polyomino can have if it spans a 128-by343 rectangle?",
        "answer": "470",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 788,
        "difficulty": 3.5
      },
      {
        "id": "a7832a91f4c17f4a3fd2a69ec9a9ded8511358c4599128878cf9d8ed8ad899e7",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=5, A C=4, B C=6$. The angle bisector of $C$ intersects side $A B$ at $X$. Points $M$ and $N$ are drawn on sides $B C$ and $A C$, respectively, such that $\\overline{X M} \\| \\overline{A C}$ and $\\overline{X N} \\| \\overline{B C}$. Compute the length $M N$.",
        "answer": "\\frac{3 \\sqrt{14}}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2368,
        "difficulty": 4.0
      },
      {
        "id": "dff2f9d36aa75940c967c613b979122569cd7825083146069ac5783a5860e230",
        "domain": "omni",
        "question": "Compute $\\sum_{n=2009}^{\\infty} \\frac{1}{\\binom{n}{2009}}$",
        "answer": "\\frac{2009}{2008}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1787,
        "difficulty": 4.0
      },
      {
        "id": "99feab514d7cceb504d5d4c5b4474edd42f04dc45be6134b0b09427a158be5f4",
        "domain": "omni",
        "question": "The point \\((p, q)\\) is on the line \\(y=\\frac{2}{5} x\\). Also, the area of the rectangle shown is 90. What is the value of \\(p\\)?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "cayley",
        "source_index": 2672,
        "difficulty": 2.5
      },
      {
        "id": "9aaffe8a65a9853a870de2b095293998db2a9c20df7c8959f2ff509166a18d09",
        "domain": "omni",
        "question": "Suppose that $f$ is a function from $\\mathbb{R}$ to $\\mathbb{R}$ such that \\[ f(x) + f\\left( 1 - \\frac{1}{x} \\right) = \\arctan x \\] for all real $x \\neq 0$. (As usual, $y = \\arctan x$ means $-\\pi/2 < y < \\pi/2$ and $\\tan y = x$.) Find \\[ \\int_0^1 f(x)\\,dx. \\]",
        "answer": "\\frac{3\\pi}{8}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "putnam",
        "source_index": 3281,
        "difficulty": 7.0
      },
      {
        "id": "015896306f3e6c345e7e3e80cd5d157165f704fb4c22d81d71b59bd384a4a79b",
        "domain": "omni",
        "question": "How many elements are in the set obtained by transforming $\\{(0,0),(2,0)\\} 14$ times?",
        "answer": "477",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 747,
        "difficulty": 4.5
      },
      {
        "id": "5aeec0ee09d5964ddfcda28d0999db88743cd7a723fce5e20e121fda275abdbf",
        "domain": "omni",
        "question": "For some integers $m$ and $n$, the expression $(x+m)(x+n)$ is equal to a quadratic expression in $x$ with a constant term of -12. Which of the following cannot be a value of $m$?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3431,
        "difficulty": 2.0
      },
      {
        "id": "cb0d8f8e6e00eeb5d35c8e96063daca180287901869fc037bf89d54000b369ab",
        "domain": "omni",
        "question": "A robotic grasshopper jumps 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After every fourth jump, the grasshopper restarts the sequence of jumps. After a total of $n$ jumps, the position of the grasshopper is 162 cm to the west and 158 cm to the south of its original position. What is the sum of the squares of the digits of $n$?",
        "answer": "22",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2991,
        "difficulty": 2.5
      },
      {
        "id": "538af85a6b2055ad7bcda4437918ddbc7ff604ca1ad32fb71839c7e41d6bbeb1",
        "domain": "omni",
        "question": "A rectangle has positive integer side lengths and an area of 24. What perimeter of the rectangle cannot be?",
        "answer": "36",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "cayley",
        "source_index": 3437,
        "difficulty": 2.0
      },
      {
        "id": "b4a9d176e3316de0a65b2990c64f9d370d7f8e97327888cfed0f56d0ece536b5",
        "domain": "omni",
        "question": "Real numbers $x, y$, and $z$ are chosen from the interval $[-1,1]$ independently and uniformly at random. What is the probability that $|x|+|y|+|z|+|x+y+z|=|x+y|+|y+z|+|z+x|$?",
        "answer": "\\frac{3}{8}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2162,
        "difficulty": 4.0
      },
      {
        "id": "aeb129c1c996aced17e752e003b94890d69ac08cdbe63da1c94df352dfb33b5d",
        "domain": "omni",
        "question": "How many solutions in nonnegative integers $(a, b, c)$ are there to the equation $2^{a}+2^{b}=c!\\quad ?$",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 533,
        "difficulty": 5.0
      },
      {
        "id": "8ed56bee0584519fce817a6e0665f65a72b49bd5d5fad5c88556566343d12fdf",
        "domain": "omni",
        "question": "Consider the following flowchart: INPUT $\\rightarrow$ Subtract $8 \\rightarrow \\square \\rightarrow$ Divide by $2 \\rightarrow \\square$ Add $16 \\rightarrow$ OUTPUT. If the OUTPUT is 32, what was the INPUT?",
        "answer": "40",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3518,
        "difficulty": 2.0
      },
      {
        "id": "5e1efcab22976272c26ef7196e69ba3f8edca51ff46aae4f4161c71f6a1af608",
        "domain": "omni",
        "question": "In 12 years, Janice will be 8 times as old as she was 2 years ago. How old is Janice now?",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3500,
        "difficulty": 1.5
      },
      {
        "id": "3aec64e095cd5c1dce4216823f6411723a29e84e706f335aeef32b305919f76a",
        "domain": "omni",
        "question": "For each \\(i \\in\\{1, \\ldots, 10\\}, a_{i}\\) is chosen independently and uniformly at random from \\([0, i^{2}]\\). Let \\(P\\) be the probability that \\(a_{1}<a_{2}<\\cdots<a_{10}\\). Estimate \\(P\\).",
        "answer": "0.003679",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2385,
        "difficulty": 5.0
      },
      {
        "id": "2931bc824a307d79903c4474393ac5c30e173e93762cc500b62e8477428c3264",
        "domain": "omni",
        "question": "Let $\\omega_{1}$ be a circle of radius 5, and let $\\omega_{2}$ be a circle of radius 2 whose center lies on $\\omega_{1}$. Let the two circles intersect at $A$ and $B$, and let the tangents to $\\omega_{2}$ at $A$ and $B$ intersect at $P$. If the area of $\\triangle ABP$ can be expressed as $\\frac{a \\sqrt{b}}{c}$, where $b$ is square-free and $a, c$ are relatively prime positive integers, compute $100a+10b+c$.",
        "answer": "19285",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2483,
        "difficulty": 5.0
      },
      {
        "id": "77cf0bafae5545e0dd6515a793ee3cb98212449402922ae98a88274b0da89f43",
        "domain": "omni",
        "question": "If $2x + 6 = 16$, what is the value of $x + 4$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2948,
        "difficulty": 1.0
      },
      {
        "id": "89eecec4051a8556e43926135886b40001f2ee46be4373635e303b9df4e45c38",
        "domain": "omni",
        "question": "Suppose $$h \\cdot a \\cdot r \\cdot v \\cdot a \\cdot r \\cdot d=m \\cdot i \\cdot t=h \\cdot m \\cdot m \\cdot t=100$$ Find $(r \\cdot a \\cdot d) \\cdot(t \\cdot r \\cdot i \\cdot v \\cdot i \\cdot a)$.",
        "answer": "10000",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2534,
        "difficulty": 4.75
      },
      {
        "id": "4d0267e12ee9215c349d8f6efcf5aca37edbb4211d803f29d8a93a3320470e10",
        "domain": "omni",
        "question": "The integer 48178 includes the block of digits 178. How many integers between 10000 and 100000 include the block of digits 178?",
        "answer": "280",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 2860,
        "difficulty": 2.0
      },
      {
        "id": "5f5604d191338c5436ad2e8a665fc45ad90c726a10753128a1620d2334369a55",
        "domain": "omni",
        "question": "How many pairs of integers $(a, b)$, with $1 \\leq a \\leq b \\leq 60$, have the property that $b$ is divisible by $a$ and $b+1$ is divisible by $a+1$?",
        "answer": "106",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Equations and Inequalities -> Other",
        "source": "HMMT_2",
        "source_index": 804,
        "difficulty": 4.0
      },
      {
        "id": "fd249488625aa5686a7c88b8160b2fe8278244176bc91c17cbc717e587064e83",
        "domain": "omni",
        "question": "Define the sequences $(a_n),(b_n)$ by\n\\begin{align*}\n& a_n, b_n > 0, \\forall n\\in\\mathbb{N_+} \\\\ \n& a_{n+1} = a_n - \\frac{1}{1+\\sum_{i=1}^n\\frac{1}{a_i}} \\\\ \n& b_{n+1} = b_n + \\frac{1}{1+\\sum_{i=1}^n\\frac{1}{b_i}}\n\\end{align*}\n1) If $a_{100}b_{100} = a_{101}b_{101}$, find the value of $a_1-b_1$;\n2) If $a_{100} = b_{99}$, determine which is larger between $a_{100}+b_{100}$ and $a_{101}+b_{101}$.",
        "answer": "199",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "china_national_olympiad",
        "source_index": 120,
        "difficulty": 9.0
      },
      {
        "id": "f2e6a980c16075e40fddf74ba6a3992b947dd94f0ba2edf662e5794cd9f6f315",
        "domain": "omni",
        "question": "Regular octagon $C H I L D R E N$ has area 1. Find the area of pentagon $C H I L D$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2606,
        "difficulty": 4.0
      },
      {
        "id": "00925ffb8b276d64872eb8ed6175a273cfd1e57ccefe828d2926bec262644722",
        "domain": "omni",
        "question": "Eight points are chosen on the circumference of a circle, labelled $P_{1}, P_{2}, \\ldots, P_{8}$ in clockwise order. A route is a sequence of at least two points $P_{a_{1}}, P_{a_{2}}, \\ldots, P_{a_{n}}$ such that if an ant were to visit these points in their given order, starting at $P_{a_{1}}$ and ending at $P_{a_{n}}$, by following $n-1$ straight line segments (each connecting each $P_{a_{i}}$ and $P_{a_{i+1}}$ ), it would never visit a point twice or cross its own path. Find the number of routes.",
        "answer": "8744",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2585,
        "difficulty": 4.0
      },
      {
        "id": "fde77d8c53731438a74c772999b310c2335547d8fbcf92474173dd3d67b55531",
        "domain": "omni",
        "question": "Let $A B C D$ be a quadrilateral inscribed in a unit circle with center $O$. Suppose that $\\angle A O B=\\angle C O D=135^{\\circ}, B C=1$. Let $B^{\\prime}$ and $C^{\\prime}$ be the reflections of $A$ across $B O$ and $C O$ respectively. Let $H_{1}$ and $H_{2}$ be the orthocenters of $A B^{\\prime} C^{\\prime}$ and $B C D$, respectively. If $M$ is the midpoint of $O H_{1}$, and $O^{\\prime}$ is the reflection of $O$ about the midpoint of $M H_{2}$, compute $O O^{\\prime}$.",
        "answer": "\\frac{1}{4}(8-\\sqrt{6}-3 \\sqrt{2})",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 660,
        "difficulty": 4.5
      },
      {
        "id": "77b2d02e9ef9fde54026c400d1b168404fcf6cbd43be15ad503af6b524e3b1dd",
        "domain": "omni",
        "question": "Let $S=\\{1,2, \\ldots, 2008\\}$. For any nonempty subset $A \\subset S$, define $m(A)$ to be the median of $A$ (when $A$ has an even number of elements, $m(A)$ is the average of the middle two elements). Determine the average of $m(A)$, when $A$ is taken over all nonempty subsets of $S$.",
        "answer": "\\frac{2009}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1685,
        "difficulty": 4.0
      },
      {
        "id": "2786175078d61bd0b6ce4fda43c7984fcd790c4b43dffc219781c29d96f5dce3",
        "domain": "omni",
        "question": "Let $A B C$ be an equilateral triangle with side length 1. Points $D, E, F$ lie inside triangle $A B C$ such that $A, E, F$ are collinear, $B, F, D$ are collinear, $C, D, E$ are collinear, and triangle $D E F$ is equilateral. Suppose that there exists a unique equilateral triangle $X Y Z$ with $X$ on side $\\overline{B C}, Y$ on side $\\overline{A B}$, and $Z$ on side $\\overline{A C}$ such that $D$ lies on side $\\overline{X Z}, E$ lies on side $\\overline{Y Z}$, and $F$ lies on side $\\overline{X Y}$. Compute $A Z$.",
        "answer": "\\frac{1}{1+\\sqrt[3]{2}}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 808,
        "difficulty": 5.5
      },
      {
        "id": "f4efc54462829f311edb8c1bb86bd8b3cc7bb91a8c1f38c3804d98bfd491fafb",
        "domain": "omni",
        "question": "Let $x, y$, and $z$ be distinct real numbers that sum to 0. Find the maximum possible value of $$\\frac{x y+y z+z x}{x^{2}+y^{2}+z^{2}}$$",
        "answer": "-1/2",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1273,
        "difficulty": 5.0
      },
      {
        "id": "1b6c66c707e722547e17cbda1c10e95e983b3371230ee4fccbc2c18e0212b076",
        "domain": "omni",
        "question": "Find the area of the region between a circle of radius 100 and a circle of radius 99.",
        "answer": "199 \\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2427,
        "difficulty": 3.5
      },
      {
        "id": "a9eb274a9807a42c53bdf2490ef1605cd372590aae30295887957facd972d63a",
        "domain": "omni",
        "question": "A pebble is shaped as the intersection of a cube of side length 1 with the solid sphere tangent to all of the cube's edges. What is the surface area of this pebble?",
        "answer": "\\frac{6 \\sqrt{2}-5}{2} \\pi",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 711,
        "difficulty": 4.5
      },
      {
        "id": "8224da27220ef5805f2466a92195778fb03d65f31194c281fc1ea14a4a5332a9",
        "domain": "omni",
        "question": "In a magic square, what is the sum \\( a+b+c \\)?",
        "answer": "47",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Other",
        "source": "pascal",
        "source_index": 2866,
        "difficulty": 1.5
      },
      {
        "id": "9624de5351c9c7abc536c4ba7702e9a5ac8905d9d774d527731709dadbda8ee0",
        "domain": "omni",
        "question": "How many different numbers are obtainable from five 5s by first concatenating some of the 5s, then multiplying them together? For example, we could do $5 \\cdot 55 \\cdot 55,555 \\cdot 55$, or 55555, but not $5 \\cdot 5$ or 2525.",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_2",
        "source_index": 2324,
        "difficulty": 4.0
      },
      {
        "id": "9e5db6416cbcff6fe2867228354b1095ef0bae875793d37d3485acc021b471d8",
        "domain": "omni",
        "question": "Two players play a game where they are each given 10 indistinguishable units that must be distributed across three locations. (Units cannot be split.) At each location, a player wins at that location if the number of units they placed there is at least 2 more than the units of the other player. If both players distribute their units randomly (i.e. there is an equal probability of them distributing their units for any attainable distribution across the 3 locations), the probability that at least one location is won by one of the players can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$.",
        "answer": "1011",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2575,
        "difficulty": 5.25
      },
      {
        "id": "edee0a459ad32f595d3cd7948c72c0f07bc41ec464a08f11401b6935a16a9ee2",
        "domain": "omni",
        "question": "Let $P R O B L E M Z$ be a regular octagon inscribed in a circle of unit radius. Diagonals $M R, O Z$ meet at $I$. Compute $L I$.",
        "answer": "\\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 988,
        "difficulty": 4.75
      },
      {
        "id": "307abe527ff309d1856cbcf438f1aebe001d3c73de33706eae4fc431d90c1a56",
        "domain": "omni",
        "question": "Find the largest possible integer $k$, such that the following statement is true:  \nLet $2009$ arbitrary non-degenerated triangles be given. In every triangle the three sides are coloured, such that one is blue, one is red and one is white. Now, for every colour separately, let us sort the lengths of the sides. We obtain\n\\[ \\left. \\begin{array}{rcl}\n & b_1 \\leq b_2\\leq\\ldots\\leq b_{2009} & \\textrm{the lengths of the blue sides }\\\\\n & r_1 \\leq r_2\\leq\\ldots\\leq r_{2009} & \\textrm{the lengths of the red sides }\\\\\n \\textrm{and } & w_1 \\leq w_2\\leq\\ldots\\leq w_{2009} & \\textrm{the lengths of the white sides }\\\\\n \\end{array}\\right.\\]\nThen there exist $k$ indices $j$ such that we can form a non-degenerated triangle with side lengths $b_j$, $r_j$, $w_j$.\n\n[i]",
        "answer": "1",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "imo_shortlist",
        "source_index": 4152,
        "difficulty": 9.5
      },
      {
        "id": "ff546b32bf35aa6ad98d2982107d4ac9dcddf6998ecc4e90ff90280481aac189",
        "domain": "omni",
        "question": "An equilateral hexagon with side length 1 has interior angles $90^{\\circ}, 120^{\\circ}, 150^{\\circ}, 90^{\\circ}, 120^{\\circ}, 150^{\\circ}$ in that order. Find its area.",
        "answer": "\\frac{3+\\sqrt{3}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2012,
        "difficulty": 4.0
      },
      {
        "id": "a6affd8f0605a9fc4057404fc39673d3c46c49250cbe57219461c4d22c71f73d",
        "domain": "omni",
        "question": "Find the largest positive integer $m$ which makes it possible to color several cells of a $70\\times 70$ table red such that [list] [*] There are no two red cells satisfying: the two rows in which they are have the same number of red cells, while the two columns in which they are also have the same number of red cells; [*] There are two rows with exactly $m$ red cells each. [/list]",
        "answer": "32",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 90,
        "difficulty": 8.0
      },
      {
        "id": "aebe76371ff72c004fcc24c178eb33fb3859bfaf543d4494f5be2c36455ed85f",
        "domain": "omni",
        "question": "Alice, Bob, and Charlie are playing a game with 6 cards numbered 1 through 6. Each player is dealt 2 cards uniformly at random. On each player's turn, they play one of their cards, and the winner is the person who plays the median of the three cards played. Charlie goes last, so Alice and Bob decide to tell their cards to each other, trying to prevent him from winning whenever possible. Compute the probability that Charlie wins regardless.",
        "answer": "\\frac{2}{15}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 687,
        "difficulty": 5.25
      },
      {
        "id": "817004d9fb7a84e34a14fb6272b86b757aece3e5836ef68ad7bb6865b1899b51",
        "domain": "omni",
        "question": "Five people take a true-or-false test with five questions. Each person randomly guesses on every question. Given that, for each question, a majority of test-takers answered it correctly, let $p$ be the probability that every person answers exactly three questions correctly. Suppose that $p=\\frac{a}{2^{b}}$ where $a$ is an odd positive integer and $b$ is a nonnegative integer. Compute 100a+b.",
        "answer": "25517",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1563,
        "difficulty": 5.0
      },
      {
        "id": "2ff833871e7e3995d45feb6a9305078118546875094e4df9de104e49fbfb6f0f",
        "domain": "omni",
        "question": "Let $d$ be a randomly chosen divisor of 2016. Find the expected value of $\\frac{d^{2}}{d^{2}+2016}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebraic Expressions -> Other",
        "source": "HMMT_11",
        "source_index": 1891,
        "difficulty": 4.0
      },
      {
        "id": "84b52c5f6c1e637e3e98c4047f6df4801b07741fcc704897b802d18256a6bdb1",
        "domain": "omni",
        "question": "A certain lottery has tickets labeled with the numbers $1,2,3, \\ldots, 1000$. The lottery is run as follows: First, a ticket is drawn at random. If the number on the ticket is odd, the drawing ends; if it is even, another ticket is randomly drawn (without replacement). If this new ticket has an odd number, the drawing ends; if it is even, another ticket is randomly drawn (again without replacement), and so forth, until an odd number is drawn. Then, every person whose ticket number was drawn (at any point in the process) wins a prize. You have ticket number 1000. What is the probability that you get a prize?",
        "answer": "1/501",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 493,
        "difficulty": 3.5
      },
      {
        "id": "8ecb25c8285acf7e417598dfa45c3af2869716a528ac4b468f3a9b429918b77c",
        "domain": "omni",
        "question": "Mr. Canada chooses a positive real $a$ uniformly at random from $(0,1]$, chooses a positive real $b$ uniformly at random from $(0,1]$, and then sets $c=a /(a+b)$. What is the probability that $c$ lies between $1 / 4$ and $3 / 4$ ?",
        "answer": "2 / 3",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 714,
        "difficulty": 3.5
      },
      {
        "id": "97778a49e655662d19ec382a452db907bead156bcc336300c9074d2a12c0bef2",
        "domain": "omni",
        "question": "What is the smallest positive integer that is a multiple of each of 3, 5, 7, and 9?",
        "answer": "315",
        "topic": "Mathematics -> Number Theory -> Least Common Multiples (LCM)",
        "source": "pascal",
        "source_index": 2921,
        "difficulty": 1.5
      },
      {
        "id": "14d39c9da97e372b38962ad0f37a4351338bca715a6a4abf07015a78f3096233",
        "domain": "omni",
        "question": "The numbers $a_{1}, a_{2}, \\ldots, a_{100}$ are a permutation of the numbers $1,2, \\ldots, 100$. Let $S_{1}=a_{1}$, $S_{2}=a_{1}+a_{2}, \\ldots, S_{100}=a_{1}+a_{2}+\\ldots+a_{100}$. What maximum number of perfect squares can be among the numbers $S_{1}, S_{2}, \\ldots, S_{100}$?",
        "answer": "60",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "izho",
        "source_index": 3368,
        "difficulty": 4.5
      },
      {
        "id": "2fbd72ea71b58b67acc11e6175f27ec14d23a695a0827dc67eb02cc43a124d70",
        "domain": "omni",
        "question": "Real numbers $a, b, c$ satisfy the equations $a+b+c=26,1 / a+1 / b+1 / c=28$. Find the value of $$\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{a}+\\frac{a}{c}+\\frac{c}{b}+\\frac{b}{a}$$",
        "answer": "725",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1171,
        "difficulty": 5.0
      },
      {
        "id": "cb2106c2875b96014b3a101f642ce3a4c31ab00aac40b539d33ce8dcbe132628",
        "domain": "omni",
        "question": "A beaver walks from $(0,0)$ to $(4,4)$ in the plane, walking one unit in the positive $x$ direction or one unit in the positive $y$ direction at each step. Moreover, he never goes to a point $(x, y)$ with $y>x$. How many different paths can he walk?",
        "answer": "14",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 370,
        "difficulty": 3.5
      },
      {
        "id": "a1bace73d8d78088feb600fe7d0abc7b5a67353f8309409505351efd3c7d1e87",
        "domain": "omni",
        "question": "Let \\omega=\\cos \\frac{2 \\pi}{727}+i \\sin \\frac{2 \\pi}{727}$. The imaginary part of the complex number $$\\prod_{k=8}^{13}\\left(1+\\omega^{3^{k-1}}+\\omega^{2 \\cdot 3^{k-1}}\\right)$$ is equal to $\\sin \\alpha$ for some angle $\\alpha$ between $-\\frac{\\pi}{2}$ and $\\frac{\\pi}{2}$, inclusive. Find $\\alpha$.",
        "answer": "\\frac{12 \\pi}{727}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_11",
        "source_index": 2000,
        "difficulty": 5.0
      },
      {
        "id": "dd5a56d171d6eb2165d42ea3f7f5657125110027194e763bba359838885cc000",
        "domain": "omni",
        "question": "Victor has a drawer with 6 socks of 3 different types: 2 complex socks, 2 synthetic socks, and 2 trigonometric socks. He repeatedly draws 2 socks at a time from the drawer at random, and stops if the socks are of the same type. However, Victor is 'synthetic-complex type-blind', so he also stops if he sees a synthetic and a complex sock. What is the probability that Victor stops with 2 socks of the same type? Assume Victor returns both socks to the drawer after each step.",
        "answer": "\\frac{3}{7}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1269,
        "difficulty": 4.0
      },
      {
        "id": "16d1a480cb7077a33d618fe4d413cc23b494b5b45e2b20700f41549c1eb25569",
        "domain": "omni",
        "question": "Compute the sum of all positive integers $n$ for which $9 \\sqrt{n}+4 \\sqrt{n+2}-3 \\sqrt{n+16}$ is an integer.",
        "answer": "18",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1865,
        "difficulty": 4.5
      },
      {
        "id": "39390aa263e0c9bd03172dd59cc05aff9aa8ae9a43ada8082cf217bedbd6e72e",
        "domain": "omni",
        "question": "Define a power cycle to be a set $S$ consisting of the nonnegative integer powers of an integer $a$, i.e. $S=\\left\\{1, a, a^{2}, \\ldots\\right\\}$ for some integer $a$. What is the minimum number of power cycles required such that given any odd integer $n$, there exists some integer $k$ in one of the power cycles such that $n \\equiv k$ $(\\bmod 1024) ?$",
        "answer": "10",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 876,
        "difficulty": 5.0
      },
      {
        "id": "83312d34ab9e28e03e610c5900a63b64bb40f6e61fc5d06e33b583c646ea4571",
        "domain": "omni",
        "question": "It is midnight on April 29th, and Abigail is listening to a song by her favorite artist while staring at her clock, which has an hour, minute, and second hand. These hands move continuously. Between two consecutive midnights, compute the number of times the hour, minute, and second hands form two equal angles and no two hands overlap.",
        "answer": "5700",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2266,
        "difficulty": 8.0
      },
      {
        "id": "42f173f720929fc64e581536739ac5414a4fdc1c4a6fee46f3e261082f784310",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that the divisors of $n$ can be partitioned into three sets with equal sums.",
        "answer": "120",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2507,
        "difficulty": 5.0
      },
      {
        "id": "d1faf12a0e2fe8a775ae28e7f3399616fcb5a4a78ee1f2de3ec769f9b7f7cf5b",
        "domain": "omni",
        "question": "Spencer is making burritos, each of which consists of one wrap and one filling. He has enough filling for up to four beef burritos and three chicken burritos. However, he only has five wraps for the burritos; in how many orders can he make exactly five burritos?",
        "answer": "25",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 570,
        "difficulty": 4.0
      },
      {
        "id": "d2efda51f54315685bc0480259c511156e342b921a6ff7bd525da2d64d5e37de",
        "domain": "omni",
        "question": "On a given circle, six points $A$ , $B$ , $C$ , $D$ , $E$ , and $F$ are chosen at random, independently and uniformly with respect to arc length. Determine the probability that the two triangles $ABC$ and $DEF$ are disjoint, i.e., have no common points.",
        "answer": "\\[\n\\frac{3}{10}\n\\]",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "usamo",
        "source_index": 168,
        "difficulty": 7.0
      },
      {
        "id": "420c209bed2525331c7004a33cbbdbe230af8c37f875c38e3934c9c90b6d861f",
        "domain": "omni",
        "question": "Let $D$ be a regular ten-sided polygon with edges of length 1. A triangle $T$ is defined by choosing three vertices of $D$ and connecting them with edges. How many different (non-congruent) triangles $T$ can be formed?",
        "answer": "8",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 692,
        "difficulty": 5.25
      },
      {
        "id": "ec5bc3a12ba2d0d38a395e1123cc9d52f813e70fad5b3077241972cf25a306b8",
        "domain": "omni",
        "question": "Let $f(x)=c x(x-1)$, where $c$ is a positive real number. We use $f^{n}(x)$ to denote the polynomial obtained by composing $f$ with itself $n$ times. For every positive integer $n$, all the roots of $f^{n}(x)$ are real. What is the smallest possible value of $c$?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1335,
        "difficulty": 5.0
      },
      {
        "id": "92b1cd9be4fa77a7022586615977eef9170ed9fc3dc5e392ab8bb09e10d5dc19",
        "domain": "omni",
        "question": "Define $\\varphi^{k}(n)$ as the number of positive integers that are less than or equal to $n / k$ and relatively prime to $n$. Find $\\phi^{2001}\\left(2002^{2}-1\\right)$. (Hint: $\\phi(2003)=2002$.)",
        "answer": "1233",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 793,
        "difficulty": 4.0
      },
      {
        "id": "ab4f75c71620bd605a63a77b79e49bb0aefc66c7fd58e4b281833804be5c5b7a",
        "domain": "omni",
        "question": "The average (mean) of a list of 10 numbers is 17. When one number is removed from the list, the new average is 16. What number was removed?",
        "answer": "26",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3546,
        "difficulty": 1.5
      },
      {
        "id": "b2a06e6d783c40f44481e4009e53842ef166a2f4cefbffe9169d3d2458d8e4ff",
        "domain": "omni",
        "question": "Find the sum of every even positive integer less than 233 not divisible by 10.",
        "answer": "10812",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1300,
        "difficulty": 2.0
      },
      {
        "id": "bd04caf5fce928cf5f670f2344497aec6475e516d935c8383309968cc7c249cc",
        "domain": "omni",
        "question": "How many real triples $(a, b, c)$ are there such that the polynomial $p(x)=x^{4}+a x^{3}+b x^{2}+a x+c$ has exactly three distinct roots, which are equal to $\\tan y, \\tan 2 y$, and $\\tan 3 y$ for some real $y$ ?",
        "answer": "18",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1465,
        "difficulty": 5.0
      },
      {
        "id": "1e172ddcd63611faea2eeb9a558171af666c256a1995e70d4f263923f749f6be",
        "domain": "omni",
        "question": "A cylinder of base radius 1 is cut into two equal parts along a plane passing through the center of the cylinder and tangent to the two base circles. Suppose that each piece's surface area is $m$ times its volume. Find the greatest lower bound for all possible values of $m$ as the height of the cylinder varies.",
        "answer": "3",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 492,
        "difficulty": 5.5
      },
      {
        "id": "ad67db185103203e5a82c878b3e7067c58ab71affb8233b0d75f2a7be13de6dd",
        "domain": "omni",
        "question": "Plot points $A, B, C$ at coordinates $(0,0),(0,1)$, and $(1,1)$ in the plane, respectively. Let $S$ denote the union of the two line segments $A B$ and $B C$. Let $X_{1}$ be the area swept out when Bobby rotates $S$ counterclockwise 45 degrees about point $A$. Let $X_{2}$ be the area swept out when Calvin rotates $S$ clockwise 45 degrees about point $A$. Find $\\frac{X_{1}+X_{2}}{2}$.",
        "answer": "\\frac{\\pi}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2183,
        "difficulty": 4.5
      },
      {
        "id": "e60391afca51368b28ac8417a54498863556675843d8fd88e413acf2084da0ab",
        "domain": "omni",
        "question": "Let $S=\\{1,2, \\ldots, 2014\\}$. For each non-empty subset $T \\subseteq S$, one of its members is chosen as its representative. Find the number of ways to assign representatives to all non-empty subsets of $S$ so that if a subset $D \\subseteq S$ is a disjoint union of non-empty subsets $A, B, C \\subseteq S$, then the representative of $D$ is also the representative of at least one of $A, B, C$.",
        "answer": "\\[ 108 \\cdot 2014! \\]",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "apmoapmo_sol",
        "source_index": 473,
        "difficulty": 7.0
      },
      {
        "id": "dc6e2af8cf15ee02f2e1a81acdb967377506a2dcef928affe3c7f057459721e1",
        "domain": "omni",
        "question": "A conical flask contains some water. When the flask is oriented so that its base is horizontal and lies at the bottom (so that the vertex is at the top), the water is 1 inch deep. When the flask is turned upside-down, so that the vertex is at the bottom, the water is 2 inches deep. What is the height of the cone?",
        "answer": "\\frac{1}{2}+\\frac{\\sqrt{93}}{6}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 823,
        "difficulty": 5.0
      },
      {
        "id": "bcffa8c1df0729d20af9a3de7623bb3b67a95ec2d01c4fde6d0615071cd61ebf",
        "domain": "omni",
        "question": "Two $4 \\times 4$ squares are randomly placed on an $8 \\times 8$ chessboard so that their sides lie along the grid lines of the board. What is the probability that the two squares overlap?",
        "answer": "529/625",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 768,
        "difficulty": 5.0
      },
      {
        "id": "38097e1b15baf13069d8a1a43d938f01b458be24f4f64c9b9caf536dcc028f85",
        "domain": "omni",
        "question": "For any positive integer $x$, define $\\operatorname{Accident}(x)$ to be the set of ordered pairs $(s, t)$ with $s \\in \\{0,2,4,5,7,9,11\\}$ and $t \\in\\{1,3,6,8,10\\}$ such that $x+s-t$ is divisible by 12. For any nonnegative integer $i$, let $a_{i}$ denote the number of $x \\in\\{0,1, \\ldots, 11\\}$ for which $|\\operatorname{Accident}(x)|=i$. Find $$a_{0}^{2}+a_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2}+a_{5}^{2}$$",
        "answer": "26",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1769,
        "difficulty": 3.5
      },
      {
        "id": "b0f3407a40e2da001772dd40194ab462f56732d71f850515f111ec19d0fc9f68",
        "domain": "omni",
        "question": "How many different combinations of 4 marbles can be made from 5 indistinguishable red marbles, 4 indistinguishable blue marbles, and 2 indistinguishable black marbles?",
        "answer": "12",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 397,
        "difficulty": 3.5
      },
      {
        "id": "7740e817b7557ace43af2929ad7c3635af510fdb5adbbf7571412c82cd2fea80",
        "domain": "omni",
        "question": "The walls of a room are in the shape of a triangle $A B C$ with $\\angle A B C=90^{\\circ}, \\angle B A C=60^{\\circ}$, and $A B=6$. Chong stands at the midpoint of $B C$ and rolls a ball toward $A B$. Suppose that the ball bounces off $A B$, then $A C$, then returns exactly to Chong. Find the length of the path of the ball.",
        "answer": "3\\sqrt{21}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 635,
        "difficulty": 4.5
      },
      {
        "id": "bc02feb87d24866c040894666ca43a55b3a76377ad785958471e2dd5cc14ee6d",
        "domain": "omni",
        "question": "If $x+\\sqrt{81}=25$, what is the value of $x$?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2675,
        "difficulty": 1.0
      },
      {
        "id": "0efba9a6a6e0fd31bc444102e2c1c9841c13d37468fa064baaeda6d6e3e9f737",
        "domain": "omni",
        "question": "The Fibonacci sequence is defined as follows: $F_{0}=0, F_{1}=1$, and $F_{n}=F_{n-1}+F_{n-2}$ for all integers $n \\geq 2$. Find the smallest positive integer $m$ such that $F_{m} \\equiv 0(\\bmod 127)$ and $F_{m+1} \\equiv 1(\\bmod 127)$.",
        "answer": "256",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "HMMT_2",
        "source_index": 1362,
        "difficulty": 5.5
      },
      {
        "id": "feec6df5955c4582dd752fc7aa14be4e3a31950471c249b70a908ed0147e4701",
        "domain": "omni",
        "question": "For positive integers $n$ and $k$, let $\\mho(n, k)$ be the number of distinct prime divisors of $n$ that are at least $k$. Find the closest integer to $$\\sum_{n=1}^{\\infty} \\sum_{k=1}^{\\infty} \\frac{\\mho(n, k)}{3^{n+k-7}}$$",
        "answer": "167",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1535,
        "difficulty": 6.0
      },
      {
        "id": "435551cbe333ed98c4e61fcb363c53f15286057b854c37b6c65ff510262dabf2",
        "domain": "omni",
        "question": "A positive integer is called fancy if it can be expressed in the form $$2^{a_{1}}+2^{a_{2}}+\\cdots+2^{a_{100}}$$ where $a_{1}, a_{2}, \\ldots, a_{100}$ are non-negative integers that are not necessarily distinct. Find the smallest positive integer $n$ such that no multiple of $n$ is a fancy number.",
        "answer": "2^{101}-1",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Number Theory -> Congruences",
        "source": "apmoapmo_sol",
        "source_index": 1635,
        "difficulty": 6.0
      },
      {
        "id": "946c5df20f85cd6f1218f758687e1aca35933deee9db4cb1356155d7e3e35abc",
        "domain": "omni",
        "question": "Find the minimum positive integer $k$ such that there exists a function $f$ from the set $\\Bbb{Z}$ of all integers to $\\{1, 2, \\ldots k\\}$ with the property that $f(x) \\neq f(y)$ whenever $|x-y| \\in \\{5, 7, 12\\}$.",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "apmo",
        "source_index": 3711,
        "difficulty": 7.5
      },
      {
        "id": "fa348ef10037f7b329c5c7103ba5ef79d11815d6283c9c3058aa593a6718adbb",
        "domain": "omni",
        "question": "Somewhere in the universe, $n$ students are taking a 10-question math competition. Their collective performance is called laughable if, for some pair of questions, there exist 57 students such that either all of them answered both questions correctly or none of them answered both questions correctly. Compute the smallest $n$ such that the performance is necessarily laughable.",
        "answer": "253",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1177,
        "difficulty": 8.0
      },
      {
        "id": "495dcf4acc840009de31328b4322d487bcdc07b8e49e12c8216b3dff84c5f993",
        "domain": "omni",
        "question": "Farmer Bill's 1000 animals - ducks, cows, and rabbits - are standing in a circle. In order to feel safe, every duck must either be standing next to at least one cow or between two rabbits. If there are 600 ducks, what is the least number of cows there can be for this to be possible?",
        "answer": "201",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1091,
        "difficulty": 5.0
      },
      {
        "id": "15d37be8e1cbcb21f264aad8973b04d0fdbaa1ae1efb39539d1d86930fa13928",
        "domain": "omni",
        "question": "Let \\mathcal{V} be the volume enclosed by the graph $x^{2016}+y^{2016}+z^{2}=2016$. Find \\mathcal{V} rounded to the nearest multiple of ten.",
        "answer": "360",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals; Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals",
        "source": "HMMT_11",
        "source_index": 1849,
        "difficulty": 4.0
      },
      {
        "id": "8796664df07cd30c6fd70ed500ef4258212810c34856fae1400cf59828850293",
        "domain": "omni",
        "question": "Five people of heights $65,66,67,68$, and 69 inches stand facing forwards in a line. How many orders are there for them to line up, if no person can stand immediately before or after someone who is exactly 1 inch taller or exactly 1 inch shorter than himself?",
        "answer": "14",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2627,
        "difficulty": 4.5
      },
      {
        "id": "9b177d3bedff88baa6922f4e28000927a6c748f6306954a40771e173c9fbb390",
        "domain": "omni",
        "question": "What is the value of $1^{3}+2^{3}+3^{3}+4^{3}$?",
        "answer": "10^{2}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3004,
        "difficulty": 1.0
      },
      {
        "id": "1b96ec631cc47a9851055c104f93924156c93edcda16054cb696586a9d4b5043",
        "domain": "omni",
        "question": "$A B C D$ is a rectangle with $A B=20$ and $B C=3$. A circle with radius 5, centered at the midpoint of $D C$, meets the rectangle at four points: $W, X, Y$, and $Z$. Find the area of quadrilateral $W X Y Z$.",
        "answer": "27",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2242,
        "difficulty": 4.0
      },
      {
        "id": "d1a57fcf974bf3df96a76f688bc8ac68cba8fb74be57abe666961b05edc82677",
        "domain": "omni",
        "question": "Evaluate the expression where the digit 2 appears 2013 times.",
        "answer": "\\frac{2013}{2014}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2223,
        "difficulty": 4.0
      },
      {
        "id": "06f2019ba8189f8ce4bf1c860d01b6cb5f642d67948e894648c3d0bca8d24f4a",
        "domain": "omni",
        "question": "Suppose that $x, y$, and $z$ are complex numbers of equal magnitude that satisfy $$x+y+z=-\\frac{\\sqrt{3}}{2}-i \\sqrt{5}$$ and $$x y z=\\sqrt{3}+i \\sqrt{5}.$$ If $x=x_{1}+i x_{2}, y=y_{1}+i y_{2}$, and $z=z_{1}+i z_{2}$ for real $x_{1}, x_{2}, y_{1}, y_{2}, z_{1}$, and $z_{2}$, then $$\\left(x_{1} x_{2}+y_{1} y_{2}+z_{1} z_{2}\\right)^{2}$$ can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "1516",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1881,
        "difficulty": 6.0
      },
      {
        "id": "af4f0cfb3108652f034fe569cda1aab3581052f5d129c297622e9358c641ea32",
        "domain": "omni",
        "question": "Let $w, x, y, z$ be real numbers such that $w+x+y+z =5$, $2 w+4 x+8 y+16 z =7$, $3 w+9 x+27 y+81 z =11$, $4 w+16 x+64 y+256 z =1$. What is the value of $5 w+25 x+125 y+625 z ?$",
        "answer": "-60",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2485,
        "difficulty": 5.0
      },
      {
        "id": "e2b87cd34bd1576d2177e2058547967987d83a2dcd4187eaad3eb60217d39626",
        "domain": "omni",
        "question": "Undecillion years ago in a galaxy far, far away, there were four space stations in the three-dimensional space, each pair spaced 1 light year away from each other. Admiral Ackbar wanted to establish a base somewhere in space such that the sum of squares of the distances from the base to each of the stations does not exceed 15 square light years. (The sizes of the space stations and the base are negligible.) Determine the volume, in cubic light years, of the set of all possible locations for the Admiral's base.",
        "answer": "\\frac{27 \\sqrt{6}}{8} \\pi",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2072,
        "difficulty": 4.0
      },
      {
        "id": "deee8c6111b701ec8a640a60b1defbbe4c380cce864d6d0b2718f38103c1b8e5",
        "domain": "omni",
        "question": "What is the minimum total number of boxes that Carley could have bought if each treat bag contains exactly 1 chocolate, 1 mint, and 1 caramel, and chocolates come in boxes of 50, mints in boxes of 40, and caramels in boxes of 25?",
        "answer": "17",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3081,
        "difficulty": 2.0
      },
      {
        "id": "76bcfdaf777f7a706c28f3b4c542e88c3af30a47f07cd3d2d58fe3f7cbb59250",
        "domain": "omni",
        "question": "Dizzy Daisy is standing on the point $(0,0)$ on the $xy$-plane and is trying to get to the point $(6,6)$. She starts facing rightward and takes a step 1 unit forward. On each subsequent second, she either takes a step 1 unit forward or turns 90 degrees counterclockwise then takes a step 1 unit forward. She may never go on a point outside the square defined by $|x| \\leq 6,|y| \\leq 6$, nor may she ever go on the same point twice. How many different paths may Daisy take?",
        "answer": "131922",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1263,
        "difficulty": 5.0
      },
      {
        "id": "76fb7de59b05b90c65965072fef8e0806d1a9d21336c4c927681c79ad6e04dfc",
        "domain": "omni",
        "question": "Find the value of $1006 \\sin \\frac{\\pi}{1006}$. Approximating directly by $\\pi=3.1415 \\ldots$ is worth only 3 points.",
        "answer": "3.1415875473",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_11",
        "source_index": 2400,
        "difficulty": 5.0
      },
      {
        "id": "85b27a3f8884a976f82348871f67f7cdb370fb6071a183f16aa8b0424de8080c",
        "domain": "omni",
        "question": "Let $\\mathcal{P}_{1}, \\mathcal{P}_{2}, \\mathcal{P}_{3}$ be pairwise distinct parabolas in the plane. Find the maximum possible number of intersections between two or more of the $\\mathcal{P}_{i}$. In other words, find the maximum number of points that can lie on two or more of the parabolas $\\mathcal{P}_{1}, \\mathcal{P}_{2}, \\mathcal{P}_{3}$.",
        "answer": "12",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Curves -> Other",
        "source": "HMMT_11",
        "source_index": 2373,
        "difficulty": 4.5
      },
      {
        "id": "13d5175688290a07cc96efabe4f32a5cadd158a8edb383c40180bb5fa87e7832",
        "domain": "omni",
        "question": "Points $A, B, C, D$ lie on a circle in that order such that $\\frac{A B}{B C}=\\frac{D A}{C D}$. If $A C=3$ and $B D=B C=4$, find $A D$.",
        "answer": "\\frac{3}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2599,
        "difficulty": 4.0
      },
      {
        "id": "f812b9e681c73fe65f9c675d197bebb92ff6036f7f28eab513a3af8cf41f4b30",
        "domain": "omni",
        "question": "A certain state issues license plates consisting of six digits (from 0 through 9). The state requires that any two plates differ in at least two places. (Thus the plates $\\boxed{027592}$ and $\\boxed{020592}$ cannot both be used.) Determine, with proof, the maximum number of distinct license plates that the state can use.",
        "answer": "\\[ 10^5 \\]",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usamo",
        "source_index": 311,
        "difficulty": 7.0
      },
      {
        "id": "377700694b5a7a9760e1ebc027ac24efeff732519c7ebc7e9a549e2bffaf71ca",
        "domain": "omni",
        "question": "In an equilateral triangle $\\triangle PRS$, if $QS=QT$ and $\\angle QTS=40^\\circ$, what is the value of $x$?",
        "answer": "80",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "fermat",
        "source_index": 2770,
        "difficulty": 2.5
      },
      {
        "id": "de64603a95d93eb3b0ceda1ae08551b004ce185c8cb3fdaec360765fddad7e08",
        "domain": "omni",
        "question": "For distinct positive integers $a, b<2012$, define $f(a, b)$ to be the number of integers $k$ with $1\\le k<2012$ such that the remainder when $ak$ divided by $2012$ is greater than that of $bk$ divided by $2012$. Let $S$ be the minimum value of $f(a, b)$, where $a$ and $b$ range over all pairs of distinct positive integers less than $2012$. Determine $S$.",
        "answer": "502",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "usajmo",
        "source_index": 3571,
        "difficulty": 6.0
      },
      {
        "id": "c4fe6cda653ed2238cab88c50f96cbd5dc2e9d4dcf65332d01b9dff7eb8a8509",
        "domain": "omni",
        "question": "Let $\\lfloor x\\rfloor$ denote the largest integer less than or equal to $x$, and let $\\{x\\}$ denote the fractional part of $x$. For example, $\\lfloor\\pi\\rfloor=3$, and $\\{\\pi\\}=0.14159 \\ldots$, while $\\lfloor 100\\rfloor=100$ and $\\{100\\}=0$. If $n$ is the largest solution to the equation $\\frac{\\lfloor n\\rfloor}{n}=\\frac{2015}{2016}$, compute $\\{n\\}$.",
        "answer": "\\frac{2014}{2015}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_11",
        "source_index": 2390,
        "difficulty": 4.0
      },
      {
        "id": "1c1260d255846d6dc71eb5594b953becd0c567c6eca58c2fb1dba32c57b24bdd",
        "domain": "omni",
        "question": "We are given triangle $A B C$, with $A B=9, A C=10$, and $B C=12$, and a point $D$ on $B C . B$ and $C$ are reflected in $A D$ to $B^{\\prime}$ and $C^{\\prime}$, respectively. Suppose that lines $B C^{\\prime}$ and $B^{\\prime} C$ never meet (i.e., are parallel and distinct). Find $B D$.",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 434,
        "difficulty": 3.5
      },
      {
        "id": "74fb5dcafe8a3b1d22291b79e0a2be296baed59cb2451c331312f30bbb283df5",
        "domain": "omni",
        "question": "A positive integer $n$ is a multiple of 7. The square root of $n$ is between 17 and 18. How many possible values of $n$ are there?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3028,
        "difficulty": 2.5
      },
      {
        "id": "baf590c0bdea890c3c00162633ffc011de72505e266996c12e5aae7109c24c37",
        "domain": "omni",
        "question": "Solve the equation $$\\sqrt{x+\\sqrt{4x+\\sqrt{16x+\\sqrt{\\ldots+\\sqrt{4^{2008}x+3}}}}}-\\sqrt{x}=1$$",
        "answer": "\\frac{1}{2^{4016}}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 3316,
        "difficulty": 5.25
      },
      {
        "id": "93267e78446e5f80d784f50236fa099e521a1f7fe8b084a83a3cab236ece8a40",
        "domain": "omni",
        "question": "Let $S$ be the set of all positive factors of 6000. What is the probability of a random quadruple $(a, b, c, d) \\in S^{4}$ satisfies $$\\operatorname{lcm}(\\operatorname{gcd}(a, b), \\operatorname{gcd}(c, d))=\\operatorname{gcd}(\\operatorname{lcm}(a, b), \\operatorname{lcm}(c, d)) ?$$",
        "answer": "\\frac{41}{512}",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 831,
        "difficulty": 5.25
      },
      {
        "id": "3f3c19665a0fa944f1707a03c1904eb4f348b67af8ca84bb7b6da23c63d22bf3",
        "domain": "omni",
        "question": "The Fibonacci numbers are defined by $F_{0}=0, F_{1}=1$, and $F_{n}=F_{n-1}+F_{n-2}$ for $n \\geq 2$. There exist unique positive integers $n_{1}, n_{2}, n_{3}, n_{4}, n_{5}, n_{6}$ such that $\\sum_{i_{1}=0}^{100} \\sum_{i_{2}=0}^{100} \\sum_{i_{3}=0}^{100} \\sum_{i_{4}=0}^{100} \\sum_{i_{5}=0}^{100} F_{i_{1}+i_{2}+i_{3}+i_{4}+i_{5}}=F_{n_{1}}-5 F_{n_{2}}+10 F_{n_{3}}-10 F_{n_{4}}+5 F_{n_{5}}-F_{n_{6}}$. Find $n_{1}+n_{2}+n_{3}+n_{4}+n_{5}+n_{6}$.",
        "answer": "1545",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 657,
        "difficulty": 5.25
      },
      {
        "id": "be57524c16566e0905fb58704ba609c717665d49418542d6627c1d36165e3d83",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. The altitude from $A$ intersects $B C$ at $D$. Let $\\omega_{1}$ and $\\omega_{2}$ be the incircles of $A B D$ and $A C D$, and let the common external tangent of $\\omega_{1}$ and $\\omega_{2}$ (other than $B C$) intersect $A D$ at $E$. Compute the length of $A E$.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2405,
        "difficulty": 5.0
      },
      {
        "id": "07366e77639feebda758fe611cc1444a00ccabe643b6a39611abc1f78a406b83",
        "domain": "omni",
        "question": "Two positive rational numbers $x$ and $y$, when written in lowest terms, have the property that the sum of their numerators is 9 and the sum of their denominators is 10 . What is the largest possible value of $x+y$ ?",
        "answer": "73 / 9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 665,
        "difficulty": 5.0
      },
      {
        "id": "35eb5058784d963d0cc4f6d01c48383aa31ed57f2161431dda9dcdfadd6c7679",
        "domain": "omni",
        "question": "Let the sequence $\\left\\{a_{i}\\right\\}_{i=0}^{\\infty}$ be defined by $a_{0}=\\frac{1}{2}$ and $a_{n}=1+\\left(a_{n-1}-1\\right)^{2}$. Find the product $$\\prod_{i=0}^{\\infty} a_{i}=a_{0} a_{1} a_{2}$$",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2494,
        "difficulty": 5.0
      },
      {
        "id": "47428e7eb06c652a18a4d7a7240cef3a1356664a29bf1befee051abb8e9cea07",
        "domain": "omni",
        "question": "How many real solutions are there to the equation $|||| x|-2|-2|-2|=|||| x|-3|-3|-3|$?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Absolute Values -> Other",
        "source": "HMMT_2",
        "source_index": 796,
        "difficulty": 5.25
      },
      {
        "id": "1ece1866772d3032bcf57742a6dda8ee8784b8b30f4b92062fd47667e307be00",
        "domain": "omni",
        "question": "Point $P$ is on the $y$-axis with $y$-coordinate greater than 0 and less than 100. A circle is drawn through $P, Q(4,4)$ and $O(0,0)$. How many possible positions for $P$ are there so that the radius of this circle is an integer?",
        "answer": "66",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "cayley",
        "source_index": 3190,
        "difficulty": 2.5
      },
      {
        "id": "4772dd619488bcd0100fdda7530f4737d813f2081ce51fa3f379c7a65daa9ea2",
        "domain": "omni",
        "question": "While waiting for their next class on Killian Court, Alesha and Belinda both write the same sequence $S$ on a piece of paper, where $S$ is a 2020-term strictly increasing geometric sequence with an integer common ratio $r$. Every second, Alesha erases the two smallest terms on her paper and replaces them with their geometric mean, while Belinda erases the two largest terms in her paper and replaces them with their geometric mean. They continue this process until Alesha is left with a single value $A$ and Belinda is left with a single value $B$. Let $r_{0}$ be the minimal value of $r$ such that $\\frac{A}{B}$ is an integer. If $d$ is the number of positive factors of $r_{0}$, what is the closest integer to $\\log _{2} d$ ?",
        "answer": "\\[\n\\boxed{2018}\n\\]",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 353,
        "difficulty": 8.0
      },
      {
        "id": "615e62eacc027ed2a1fb9c858d255753de64a11e0a0cd350af1c9c4936fbda93",
        "domain": "omni",
        "question": "The vertices of a regular hexagon are labeled $\\cos (\\theta), \\cos (2 \\theta), \\ldots, \\cos (6 \\theta)$. For every pair of vertices, Bob draws a blue line through the vertices if one of these functions can be expressed as a polynomial function of the other (that holds for all real $\\theta$ ), and otherwise Roberta draws a red line through the vertices. In the resulting graph, how many triangles whose vertices lie on the hexagon have at least one red and at least one blue edge?",
        "answer": "14",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1499,
        "difficulty": 5.0
      },
      {
        "id": "713a75e4946211c9db1ab6bc8c0f16805919c7318a615d9ee5db9329c72db92c",
        "domain": "omni",
        "question": "How many 3-digit positive integers have exactly one even digit?",
        "answer": "350",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 2679,
        "difficulty": 2.5
      },
      {
        "id": "5fe6e34c3f9ff91039504f25b568875a21d5986667b1ddd517dafaf5836ef91c",
        "domain": "omni",
        "question": "There are two values of $k$ for which the equation $x^{2}+2kx+7k-10=0$ has two equal real roots (that is, has exactly one solution for $x$). What is the sum of these values of $k$?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "fermat",
        "source_index": 2778,
        "difficulty": 3.0
      },
      {
        "id": "0fd674b13827b654e083697749930d3057cfbf8d52d04b4b685c3d8e7cb6923f",
        "domain": "omni",
        "question": "Mark and William are playing a game with a stored value. On his turn, a player may either multiply the stored value by 2 and add 1 or he may multiply the stored value by 4 and add 3. The first player to make the stored value exceed $2^{100}$ wins. The stored value starts at 1 and Mark goes first. Assuming both players play optimally, what is the maximum number of times that William can make a move?",
        "answer": "33",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_11",
        "source_index": 2170,
        "difficulty": 5.0
      },
      {
        "id": "dde6ebbce3927b3baa4e81e7223d108b2c273d268662cda9d6a6a488e147a0f4",
        "domain": "omni",
        "question": "How many times does 24 divide into 100! (factorial)?",
        "answer": "32",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1393,
        "difficulty": 4.0
      },
      {
        "id": "4ebface1c0971b01057cab4ac6a768e9186fa1241b3ab769ffa4bf6a6404d4bc",
        "domain": "omni",
        "question": "Today, Ivan the Confessor prefers continuous functions \\(f:[0,1] \\rightarrow \\mathbb{R}\\) satisfying \\(f(x)+f(y) \\geq|x-y|\\) for all pairs \\(x, y \\in[0,1]\\). Find the minimum of \\(\\int_{0}^{1} f\\) over all preferred functions.",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals",
        "source": "imc",
        "source_index": 2649,
        "difficulty": 6.0
      },
      {
        "id": "be0a48386dd044ad5551441998daee5cc9b7e1e5021ea98b2b1b213d05c36735",
        "domain": "omni",
        "question": "Let $A,B,C,D$ denote four points in space such that at most one of the distances $AB,AC,AD,BC,BD,CD$ is greater than $1$ . Determine the maximum value of the sum of the six distances.",
        "answer": "\\[ 5 + \\sqrt{3} \\]",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "usamo",
        "source_index": 166,
        "difficulty": 7.0
      },
      {
        "id": "0ca336b222e99c2675a09dc065787c087bbb6ba2ea2476e507b15c1fa044fb8e",
        "domain": "omni",
        "question": "You have infinitely many boxes, and you randomly put 3 balls into them. The boxes are labeled $1,2, \\ldots$. Each ball has probability $1 / 2^{n}$ of being put into box $n$. The balls are placed independently of each other. What is the probability that some box will contain at least 2 balls?",
        "answer": "5 / 7",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1449,
        "difficulty": 3.0
      },
      {
        "id": "d0eac682f2da40477b56985472c222427076515c0141a6c9c916ab0bab4cce8e",
        "domain": "omni",
        "question": "In triangle \\(A B C, A B=6, B C=7\\) and \\(C A=8\\). Let \\(D, E, F\\) be the midpoints of sides \\(B C\\), \\(A C, A B\\), respectively. Also let \\(O_{A}, O_{B}, O_{C}\\) be the circumcenters of triangles \\(A F D, B D E\\), and \\(C E F\\), respectively. Find the area of triangle \\(O_{A} O_{B} O_{C}\\).",
        "answer": "\\frac{21 \\sqrt{15}}{16}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1811,
        "difficulty": 4.75
      },
      {
        "id": "082131d4dcb81309e9ca0d03e3565772523b39d01a0d0c7b626c60338d707b46",
        "domain": "omni",
        "question": "How many ways are there to color the vertices of a triangle red, green, blue, or yellow such that no two vertices have the same color? Rotations and reflections are considered distinct.",
        "answer": "24",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2043,
        "difficulty": 4.0
      },
      {
        "id": "57b91fc6fd04e387849715294d093cc10db11800e2d28b5758a03e351affe547",
        "domain": "omni",
        "question": "Let $S_{7}$ denote all the permutations of $1,2, \\ldots, 7$. For any \\pi \\in S_{7}$, let $f(\\pi)$ be the smallest positive integer $i$ such that \\pi(1), \\pi(2), \\ldots, \\pi(i)$ is a permutation of $1,2, \\ldots, i$. Compute \\sum_{\\pi \\in S_{7}} f(\\pi)$.",
        "answer": "29093",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 783,
        "difficulty": 5.0
      },
      {
        "id": "6c0079e524f1a2a1f63f664347eb2f4320f2013524fd53b4ee20debe5791c239",
        "domain": "omni",
        "question": "How many integers $n$ in the set $\\{4,9,14,19, \\ldots, 2014\\}$ have the property that the sum of the decimal digits of $n$ is even?",
        "answer": "201",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2155,
        "difficulty": 4.5
      },
      {
        "id": "d454c1afbee745c02627b535b66a5ee99112bca74413598749b82f47af0f068a",
        "domain": "omni",
        "question": "Kelvin the frog currently sits at $(0,0)$ in the coordinate plane. If Kelvin is at $(x, y)$, either he can walk to any of $(x, y+1),(x+1, y)$, or $(x+1, y+1)$, or he can jump to any of $(x, y+2),(x+2, y)$ or $(x+1, y+1)$. Walking and jumping from $(x, y)$ to $(x+1, y+1)$ are considered distinct actions. Compute the number of ways Kelvin can reach $(6,8)$.",
        "answer": "1831830",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 704,
        "difficulty": 5.0
      },
      {
        "id": "18116ca1dc6393419dadbfd2155e8eedbca888e3a4aa70d08b4bb1eb02bd8850",
        "domain": "omni",
        "question": "$x, y$ are positive real numbers such that $x+y^{2}=x y$. What is the smallest possible value of $x$?",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 902,
        "difficulty": 3.5
      },
      {
        "id": "46b3de9a26a981f5c82f8f8f68140ff4dad2c6f08020942f061b5e8dddf115ef",
        "domain": "omni",
        "question": "Let $ABCDE$ be a convex pentagon such that $AB=AE=CD=1$ , $\\angle ABC=\\angle DEA=90^\\circ$ and $BC+DE=1$ . Compute the area of the pentagon.",
        "answer": "\\[ 1 \\]",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "jbmo",
        "source_index": 165,
        "difficulty": 4.5
      },
      {
        "id": "d3efbe2799cf615dbc9e0d799b535ad10b4453f487c64e622b2b272859facb08",
        "domain": "omni",
        "question": "Sylvia chose positive integers $a, b$ and $c$. Peter determined the value of $a + \\frac{b}{c}$ and got an answer of 101. Paul determined the value of $\\frac{a}{c} + b$ and got an answer of 68. Mary determined the value of $\\frac{a + b}{c}$ and got an answer of $k$. What is the value of $k$?",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2772,
        "difficulty": 2.5
      },
      {
        "id": "8cba1b0251fb3d3e4587fdd120ea9af40fd1892ea2a1524b03125a0ef4b79521",
        "domain": "omni",
        "question": "There are 20 students in a class. In total, 10 of them have black hair, 5 of them wear glasses, and 3 of them both have black hair and wear glasses. How many of the students have black hair but do not wear glasses?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3165,
        "difficulty": 1.0
      },
      {
        "id": "80ee77eb137c9a6c8bd0f8c9b8050fd530fffc919f0a426aa72761e5bd4b1cb9",
        "domain": "omni",
        "question": "Determine the number of unordered triples of distinct points in the $4 \\times 4 \\times 4$ lattice grid $\\{0,1,2,3\\}^{3}$ that are collinear in $\\mathbb{R}^{3}$ (i.e. there exists a line passing through the three points).",
        "answer": "376",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 838,
        "difficulty": 5.0
      },
      {
        "id": "1726e81e8bf2cccfd57c61cc407cfbfec7d99bedf4e634f226270e37fa3b34b6",
        "domain": "omni",
        "question": "Ten gangsters are standing on a flat surface, and the distances between them are all distinct. At twelve o’clock, when the church bells start chiming, each of them fatally shoots the one among the other nine gangsters who is the nearest. At least how many gangsters will be killed?",
        "answer": "7",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "imo_shortlist",
        "source_index": 4170,
        "difficulty": 7.0
      },
      {
        "id": "7be0e77fcbafb410b4bbec5ce6d891cf9ce309927d611db486a1f10e7485b83e",
        "domain": "omni",
        "question": "Find all integers $n$, $n \\ge 1$, such that $n \\cdot 2^{n+1}+1$ is a perfect square.",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "junior_balkan_mo",
        "source_index": 3799,
        "difficulty": 5.0
      },
      {
        "id": "7c92b7bcaee5f173e1bb97e9fddbd74af231f32224a85a239e6ff5d9d6eca7ad",
        "domain": "omni",
        "question": "Which of the following is equal to $9^{4}$?",
        "answer": "3^{8}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "cayley",
        "source_index": 3391,
        "difficulty": 2.0
      },
      {
        "id": "849577107de6234f1eb6cedc035f3be60fce719d8f1f8d211680aed49f794b16",
        "domain": "omni",
        "question": "Matt has somewhere between 1000 and 2000 pieces of paper he's trying to divide into piles of the same size (but not all in one pile or piles of one sheet each). He tries $2,3,4,5,6,7$, and 8 piles but ends up with one sheet left over each time. How many piles does he need?",
        "answer": "41",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1256,
        "difficulty": 5.0
      },
      {
        "id": "665be976ce928d806294e42c7dcda29515987ddaca7c66c312cdcf16c8951ff9",
        "domain": "omni",
        "question": "A sequence $s_{0}, s_{1}, s_{2}, s_{3}, \\ldots$ is defined by $s_{0}=s_{1}=1$ and, for every positive integer $n, s_{2 n}=s_{n}, s_{4 n+1}=s_{2 n+1}, s_{4 n-1}=s_{2 n-1}+s_{2 n-1}^{2} / s_{n-1}$. What is the value of $s_{1000}$?",
        "answer": "720",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 785,
        "difficulty": 5.0
      },
      {
        "id": "fea5fcad15b70facd20006f30c16f58ee564b22ce2341d861d3483e68218ecd6",
        "domain": "omni",
        "question": "Mike rides his bicycle at a constant speed of $30 \\mathrm{~km} / \\mathrm{h}$. How many kilometres does Mike travel in 20 minutes?",
        "answer": "10",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2929,
        "difficulty": 1.5
      },
      {
        "id": "3d9fe6e3d074183eae0cb4a7b77e6e43f9be86409f6b70bf3d79acab783d256d",
        "domain": "omni",
        "question": "Erin walks $\\frac{3}{5}$ of the way home in 30 minutes. If she continues to walk at the same rate, how many minutes will it take her to walk the rest of the way home?",
        "answer": "20",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2916,
        "difficulty": 1.0
      },
      {
        "id": "a8df59d586221a255c18eb208c0525710b21b48b91d08190938ca6cd3d9b42ce",
        "domain": "omni",
        "question": "On an $8 \\times 8$ chessboard, 6 black rooks and $k$ white rooks are placed on different cells so that each rook only attacks rooks of the opposite color. Compute the maximum possible value of $k$.",
        "answer": "14",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2341,
        "difficulty": 4.5
      },
      {
        "id": "70e5f4891cdccfb30f761192747f5cf24ae1d23aa3edef231232b0b968372efb",
        "domain": "omni",
        "question": "If $x = -3$, what is the value of $(x-3)^{2}$?",
        "answer": "36",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2709,
        "difficulty": 1.0
      },
      {
        "id": "3e22e7d47cdbea60045e1dd20143152fa95c51a59bfb0e91ecff90022396de42",
        "domain": "omni",
        "question": "A circle with center at $O$ has radius 1. Points $P$ and $Q$ outside the circle are placed such that $P Q$ passes through $O$. Tangent lines to the circle through $P$ hit the circle at $P_{1}$ and $P_{2}$, and tangent lines to the circle through $Q$ hit the circle at $Q_{1}$ and $Q_{2}$. If $\\angle P_{1} P P_{2}=45^{\\circ}$ and angle $Q_{1} Q Q_{2}=30^{\\circ}$, find the minimum possible length of arc $P_{2} Q_{2}$.",
        "answer": "\\frac{\\pi}{12}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 368,
        "difficulty": 4.5
      },
      {
        "id": "97272c5d264eda67e586933baf7cad3873fc87a19527efa0cf61049879916e30",
        "domain": "omni",
        "question": "In acute $\\triangle A B C$ with centroid $G, A B=22$ and $A C=19$. Let $E$ and $F$ be the feet of the altitudes from $B$ and $C$ to $A C$ and $A B$ respectively. Let $G^{\\prime}$ be the reflection of $G$ over $B C$. If $E, F, G$, and $G^{\\prime}$ lie on a circle, compute $B C$.",
        "answer": "13",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2004,
        "difficulty": 5.25
      },
      {
        "id": "0ffc0b30581148c60ab22a7819c07110e47aebecaa295aab77c031540b8ea758",
        "domain": "omni",
        "question": "Calculate the value of $\\sqrt{\\frac{\\sqrt{81} + \\sqrt{81}}{2}}$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 3027,
        "difficulty": 1.5
      },
      {
        "id": "df2e7dd6951f5e2a05213e37e6a92ceb630bd80d04fc75c73e183a0d08bd1236",
        "domain": "omni",
        "question": "Let $A B C D$ be a rectangle with $A B=20$ and $A D=23$. Let $M$ be the midpoint of $C D$, and let $X$ be the reflection of $M$ across point $A$. Compute the area of triangle $X B D$.",
        "answer": "575",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1847,
        "difficulty": 5.0
      },
      {
        "id": "dc4478c4d2df215d6e463b9f78e834230f19f1e5e7ff0e92bc915ab090342abd",
        "domain": "omni",
        "question": "Consider a $6 \\times 6$ grid of squares. Edmond chooses four of these squares uniformly at random. What is the probability that the centers of these four squares form a square?",
        "answer": "\\frac{1}{561}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2080,
        "difficulty": 4.0
      },
      {
        "id": "45d762ab436a9acc932d6774f1f82529d91a06bed85f42fb894e70abdb690123",
        "domain": "omni",
        "question": "Five cards labeled A, B, C, D, and E are placed consecutively in a row. How many ways can they be re-arranged so that no card is moved more than one position away from where it started?",
        "answer": "8",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1659,
        "difficulty": 4.0
      },
      {
        "id": "663c068b7f96fcd394457eb6514fe7b790d7d49e367fd0b09d8a648f19efa537",
        "domain": "omni",
        "question": "A regular tetrahedron has a square shadow of area 16 when projected onto a flat surface (light is shone perpendicular onto the plane). Compute the sidelength of the regular tetrahedron.",
        "answer": "4 \\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2320,
        "difficulty": 4.0
      },
      {
        "id": "a12ab69df0ffe80842f628cd8d6825240ad590671ed224e2b443eec7bef067f6",
        "domain": "omni",
        "question": "How many pairs of positive integers $(x, y)$ have the property that the ratio $x: 4$ equals the ratio $9: y$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Number Theory -> Integer Factorization -> Other",
        "source": "pascal",
        "source_index": 2862,
        "difficulty": 1.5
      },
      {
        "id": "53889c276ee65d1aa1fdd1400c83fe9e0a966d85e6c5c9c913d0e99df7b226c2",
        "domain": "omni",
        "question": "Suppose $x$ satisfies $x^{3}+x^{2}+x+1=0$. What are all possible values of $x^{4}+2 x^{3}+2 x^{2}+2 x+1 ?$",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 373,
        "difficulty": 3.5
      },
      {
        "id": "6dc7069c583c1cd3a77a7edd3cce6e71f33cf82ab73c15616eef12af6096c7a8",
        "domain": "omni",
        "question": "Chords $\\overline{A B}$ and $\\overline{C D}$ of circle $\\omega$ intersect at $E$ such that $A E=8, B E=2, C D=10$, and $\\angle A E C=90^{\\circ}$. Let $R$ be a rectangle inside $\\omega$ with sides parallel to $\\overline{A B}$ and $\\overline{C D}$, such that no point in the interior of $R$ lies on $\\overline{A B}, \\overline{C D}$, or the boundary of $\\omega$. What is the maximum possible area of $R$?",
        "answer": "26+6 \\sqrt{17}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2379,
        "difficulty": 5.25
      },
      {
        "id": "25221ed33c5916fa21a223e3391db725a1a469f07f56ed30cbd9732c66282388",
        "domain": "omni",
        "question": "Fifty numbers have an average of 76. Forty of these numbers have an average of 80. What is the average of the other ten numbers?",
        "answer": "60",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Averages -> Other",
        "source": "fermat",
        "source_index": 3201,
        "difficulty": 1.5
      },
      {
        "id": "447f542be4583ccb52890f825a13fb436a68edfdd24625aabb183cd783cbd2e3",
        "domain": "omni",
        "question": "Evaluate $\\sum_{i=1}^{\\infty} \\frac{(i+1)(i+2)(i+3)}{(-2)^{i}}$.",
        "answer": "96",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Series -> Other",
        "source": "HMMT_2",
        "source_index": 412,
        "difficulty": 4.0
      },
      {
        "id": "20da36eff45784d6095acbf951229379bf3f70291cb491d097066fecd6abd8c8",
        "domain": "omni",
        "question": "A $5 \\times 5$ table is called regular if each of its cells contains one of four pairwise distinct real numbers, such that each of them occurs exactly once in every $2 \\times 2$ subtable.The sum of all numbers of a regular table is called the total sum of the table. With any four numbers, one constructs all possible regular tables, computes their total sums, and counts the distinct outcomes. Determine the maximum possible count.",
        "answer": "\\boxed{60}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "jbmo",
        "source_index": 161,
        "difficulty": 5.0
      },
      {
        "id": "f7acb50de4e25310ab073a711ad4576be8539475336a4731acc202277cfc1cad",
        "domain": "omni",
        "question": "Mary and Sally were once the same height. Since then, Sally grew \\( 20\\% \\) taller and Mary's height increased by half as many centimetres as Sally's height increased. Sally is now 180 cm tall. How tall, in cm, is Mary now?",
        "answer": "165",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3402,
        "difficulty": 2.5
      },
      {
        "id": "8c6e420b32c0f7566a697a0b2eb996cac1ea3a002c4aea4b8e1c01580a44c27f",
        "domain": "omni",
        "question": "Suppose that $x$ and $y$ are real numbers with $-4 \\leq x \\leq -2$ and $2 \\leq y \\leq 4$. What is the greatest possible value of $\\frac{x+y}{x}$?",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "fermat",
        "source_index": 3117,
        "difficulty": 2.5
      },
      {
        "id": "de3aded83fd92bc16b6cfaf56d7e52ca83e48f06488d49213d0a1d62cc33dadf",
        "domain": "omni",
        "question": "What are the last 8 digits of $$11 \\times 101 \\times 1001 \\times 10001 \\times 100001 \\times 1000001 \\times 111 ?$$",
        "answer": "19754321",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 2327,
        "difficulty": 5.0
      },
      {
        "id": "31161dd63292a74daaa4c6627bb1ad3bc5c09553cf6e09b001575eabbe6465c1",
        "domain": "omni",
        "question": "Rachel has the number 1000 in her hands. When she puts the number $x$ in her left pocket, the number changes to $x+1$. When she puts the number $x$ in her right pocket, the number changes to $x^{-1}$. Each minute, she flips a fair coin. If it lands heads, she puts the number into her left pocket, and if it lands tails, she puts it into her right pocket. She then takes the new number out of her pocket. If the expected value of the number in Rachel's hands after eight minutes is $E$, then compute $\\left\\lfloor\\frac{E}{10}\\right\\rfloor$.",
        "answer": "13",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1598,
        "difficulty": 5.0
      },
      {
        "id": "0bffdd3ab143d18d52f611c9b9c10cef8735a2075b681da1ea60085cef40af75",
        "domain": "omni",
        "question": "Kelvin the Frog likes numbers whose digits strictly decrease, but numbers that violate this condition in at most one place are good enough. In other words, if $d_{i}$ denotes the $i$ th digit, then $d_{i} \\leq d_{i+1}$ for at most one value of $i$. For example, Kelvin likes the numbers 43210, 132, and 3, but not the numbers 1337 and 123. How many 5-digit numbers does Kelvin like?",
        "answer": "14034",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1506,
        "difficulty": 5.5
      },
      {
        "id": "70541437ba8ff92c7034ac313cb472036279acf0f7a6cae700bc39ae889f267e",
        "domain": "omni",
        "question": "Carrie sends five text messages to her brother each Saturday and Sunday, and two messages on other days. Over four weeks, how many text messages does Carrie send?",
        "answer": "80",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2841,
        "difficulty": 1.0
      },
      {
        "id": "9b8b74e99c15affcf165c155dd0cfe730177727ed42dedff813658bf864ea6b9",
        "domain": "omni",
        "question": "How many 48-tuples of positive integers \\(\\left(a_{1}, a_{2}, \\ldots, a_{48}\\right)\\) between 0 and 100 inclusive have the property that for all \\(1 \\leq i<j \\leq 48, a_{i} \\notin\\left\\{a_{j}, a_{j}+1\\right\\}\\) ?",
        "answer": "54^{48}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1742,
        "difficulty": 5.5
      },
      {
        "id": "c054a9c33d9e7f413a24915ad943562fb850138626fde0250b3921b2a2b07f99",
        "domain": "omni",
        "question": "Compute the surface area of a cube inscribed in a sphere of surface area $\\pi$.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 392,
        "difficulty": 3.5
      },
      {
        "id": "3f8a6f211bdd155d644b126ca4f0ba76e0ec054998b46dcfd9828a6050af9558",
        "domain": "omni",
        "question": "What is the area of the region \\( \\mathcal{R} \\) formed by all points \\( L(r, t) \\) with \\( 0 \\leq r \\leq 10 \\) and \\( 0 \\leq t \\leq 10 \\) such that the area of triangle \\( JKL \\) is less than or equal to 10, where \\( J(2,7) \\) and \\( K(5,3) \\)?",
        "answer": "\\frac{313}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area; Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3194,
        "difficulty": 2.0
      },
      {
        "id": "bde323a798cf963bec9f5f1860d1d22ef772be0efec99c20134404e40fd97bd2",
        "domain": "omni",
        "question": "A random binary string of length 1000 is chosen. Let \\(L\\) be the expected length of its longest (contiguous) palindromic substring. Estimate \\(L\\).",
        "answer": "23.120",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2441,
        "difficulty": 5.0
      },
      {
        "id": "94ec60371b6055f0296e637abcd87586a78d8cf0bf282569c4c01cedfe1b973d",
        "domain": "omni",
        "question": "A knight begins on the lower-left square of a standard chessboard. How many squares could the knight end up at after exactly 2009 legal knight's moves?",
        "answer": "32",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1782,
        "difficulty": 5.0
      },
      {
        "id": "1f19eb3a0eb4f3c73c025b75a44b068b99e522abc927f4637eea758e5d63251b",
        "domain": "omni",
        "question": "Let $a,b,c,d$ be real numbers such that $b-d \\ge 5$ and all zeros $x_1, x_2, x_3,$ and $x_4$ of the polynomial $P(x)=x^4+ax^3+bx^2+cx+d$ are real. Find the smallest value the product $(x_1^2+1)(x_2^2+1)(x_3^2+1)(x_4^2+1)$ can take.",
        "answer": "\\boxed{16}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "usamo",
        "source_index": 433,
        "difficulty": 8.0
      },
      {
        "id": "1f7193e51ec25ea9d26b6451e3b63bcd13e339a06d47ff5fe037468e95a8cf28",
        "domain": "omni",
        "question": "Let $ABC$ be an equilateral triangle of side length 6 inscribed in a circle $\\omega$. Let $A_{1}, A_{2}$ be the points (distinct from $A$) where the lines through $A$ passing through the two trisection points of $BC$ meet $\\omega$. Define $B_{1}, B_{2}, C_{1}, C_{2}$ similarly. Given that $A_{1}, A_{2}, B_{1}, B_{2}, C_{1}, C_{2}$ appear on $\\omega$ in that order, find the area of hexagon $A_{1}A_{2}B_{1}B_{2}C_{1}C_{2}$.",
        "answer": "\\frac{846\\sqrt{3}}{49}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 884,
        "difficulty": 4.5
      },
      {
        "id": "7f402f5a527dce6b4fe86525e2f6f80a5b942dbcd91b99b384f532b902a65661",
        "domain": "omni",
        "question": "Let $A B C$ be an isosceles triangle with $A B=A C$. Let $D$ and $E$ be the midpoints of segments $A B$ and $A C$, respectively. Suppose that there exists a point $F$ on ray $\\overrightarrow{D E}$ outside of $A B C$ such that triangle $B F A$ is similar to triangle $A B C$. Compute $\\frac{A B}{B C}$.",
        "answer": "\\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2185,
        "difficulty": 4.0
      },
      {
        "id": "2579dc6d72dbf3ecaa4b0d563f152bf008fcbc24b21d8c9d0d00f6ca5cb545ed",
        "domain": "omni",
        "question": "Now a ball is launched from a vertex of an equilateral triangle with side length 5. It strikes the opposite side after traveling a distance of $\\sqrt{19}$. How many times does the ball bounce before it returns to a vertex? (The final contact with a vertex does not count as a bounce.)",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1763,
        "difficulty": 5.0
      },
      {
        "id": "9d7232baacd1c6bef58173852c9eca9341049f41f9b3d610b3b2aeff0b2c00a0",
        "domain": "omni",
        "question": "Given that $\\sin A+\\sin B=1$ and $\\cos A+\\cos B=3 / 2$, what is the value of $\\cos (A-B)$?",
        "answer": "5/8",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1599,
        "difficulty": 3.5
      },
      {
        "id": "931325000d940887d4f6bfa1199e2a853d7b15432f62e139b37a7532eae717a7",
        "domain": "omni",
        "question": "Which of the following numbers is less than $\\frac{1}{20}$?",
        "answer": "\\frac{1}{25}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3509,
        "difficulty": 1.5
      },
      {
        "id": "760c277917c88f0653ceb8d0546237830c0f3f2d15002184411404fc71451f28",
        "domain": "omni",
        "question": "The game of Penta is played with teams of five players each, and there are five roles the players can play. Each of the five players chooses two of five roles they wish to play. If each player chooses their roles randomly, what is the probability that each role will have exactly two players?",
        "answer": "\\frac{51}{2500}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 681,
        "difficulty": 5.25
      },
      {
        "id": "bf78e7ce44da935def5740b18d4a936c2cacdd45aab23b8ef685f9540302d444",
        "domain": "omni",
        "question": "Evaluate $\\frac{2016!^{2}}{2015!2017!}$. Here $n$ ! denotes $1 \\times 2 \\times \\cdots \\times n$.",
        "answer": "\\frac{2016}{2017}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Factorials -> Other",
        "source": "HMMT_11",
        "source_index": 1986,
        "difficulty": 3.5
      },
      {
        "id": "1badd6d7b8fba39312f1c35b62fbaabe6d0ba11eeff42d6093c22e7265c1826d",
        "domain": "omni",
        "question": "Compute the value of $\\sqrt{105^{3}-104^{3}}$, given that it is a positive integer.",
        "answer": "181",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 499,
        "difficulty": 4.75
      },
      {
        "id": "ccb2b3167c045dd7379302ac13755226a87d6bf5d9ff8ba6cf0e10e48a401f89",
        "domain": "omni",
        "question": "Ali wants to move from point $A$ to point $B$. He cannot walk inside the black areas but he is free to move in any direction inside the white areas (not only the grid lines but the whole plane). Help Ali to find the shortest path between $A$ and $B$. Only draw the path and write its length.\n[img]https://1.bp.blogspot.com/-nZrxJLfIAp8/W1RyCdnhl3I/AAAAAAAAIzQ/NM3t5EtJWMcWQS0ig0IghSo54DQUBH5hwCK4BGAYYCw/s1600/igo%2B2016.el1.png[/img]\nby Morteza Saghafian",
        "answer": "7 + 5\\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "iranian_geometry_olympiad",
        "source_index": 3660,
        "difficulty": 6.0
      },
      {
        "id": "7e395e1a820d442645c083a18cb52683aae7de81ff8b8a784d28499e06ecfd38",
        "domain": "omni",
        "question": "A rectangle has length 8 cm and width $\\pi$ cm. A semi-circle has the same area as the rectangle. What is its radius?",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "fermat",
        "source_index": 2859,
        "difficulty": 2.5
      },
      {
        "id": "195853949852468a17e274b43b9167912907ea13b7964c874fc147e278d3733c",
        "domain": "omni",
        "question": "Let $P$ be a point selected uniformly at random in the cube $[0,1]^{3}$. The plane parallel to $x+y+z=0$ passing through $P$ intersects the cube in a two-dimensional region $\\mathcal{R}$. Let $t$ be the expected value of the perimeter of $\\mathcal{R}$. If $t^{2}$ can be written as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers, compute $100 a+b$.",
        "answer": "12108",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 880,
        "difficulty": 3.5
      },
      {
        "id": "f06c198e748a5df64a8baf95b4d0024ad668da5cbd27b99c1c0dbe4fa15ebc50",
        "domain": "omni",
        "question": "How many of the first 1000 positive integers can be written as the sum of finitely many distinct numbers from the sequence $3^{0}, 3^{1}, 3^{2}, \\ldots$?",
        "answer": "105",
        "topic": "Mathematics -> Number Theory -> Base Representations -> Other",
        "source": "HMMT_11",
        "source_index": 2275,
        "difficulty": 4.0
      },
      {
        "id": "f5b5e5a082e4ca174d4fdda7205167e680790ed7ea7ff1b5bdce42fb74932ce9",
        "domain": "omni",
        "question": "Consider five-dimensional Cartesian space $\\mathbb{R}^{5}=\\left\\{\\left(x_{1}, x_{2}, x_{3}, x_{4}, x_{5}\\right) \\mid x_{i} \\in \\mathbb{R}\\right\\}$ and consider the hyperplanes with the following equations: - $x_{i}=x_{j}$ for every $1 \\leq i<j \\leq 5$; - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=-1$ - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=0$ - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=1$. Into how many regions do these hyperplanes divide $\\mathbb{R}^{5}$ ?",
        "answer": "480",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Other",
        "source": "HMMT_11",
        "source_index": 1890,
        "difficulty": 4.5
      },
      {
        "id": "26206ca26031683721fdbc28d900c3ad70cb896d26e2d7108c84077d491ae416",
        "domain": "omni",
        "question": "Compute the number of ordered pairs of positive integers $(a, b)$ satisfying the equation $\\operatorname{gcd}(a, b) \\cdot a+b^{2}=10000$",
        "answer": "99",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 2017,
        "difficulty": 4.5
      },
      {
        "id": "39d8b318a6e13f087109e36ea4f4b0b936d79411e108f713543871f46c78d250",
        "domain": "omni",
        "question": "On Monday, Mukesh travelled \\(x \\mathrm{~km}\\) at a constant speed of \\(90 \\mathrm{~km} / \\mathrm{h}\\). On Tuesday, he travelled on the same route at a constant speed of \\(120 \\mathrm{~km} / \\mathrm{h}\\). His trip on Tuesday took 16 minutes less than his trip on Monday. What is the value of \\(x\\)?",
        "answer": "96",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "pascal",
        "source_index": 2988,
        "difficulty": 2.0
      },
      {
        "id": "f0dd7cdd0bcbaef15b52f9ffd72e16abeb525560ef936d4abe0424ab1e983ea9",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=8, B C=15$, and $A C=17$. Point $X$ is chosen at random on line segment $A B$. Point $Y$ is chosen at random on line segment $B C$. Point $Z$ is chosen at random on line segment $C A$. What is the expected area of triangle $X Y Z$ ?",
        "answer": "15",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_2",
        "source_index": 2431,
        "difficulty": 5.0
      },
      {
        "id": "34a609e7515f5a38c48c03cb1037e5ba724a403306bfca307ef05d47d3493a06",
        "domain": "omni",
        "question": "Let $\\omega$ be a circle of radius 1 centered at $O$. Let $B$ be a point on $\\omega$, and let $l$ be the line tangent to $\\omega$ at $B$. Let $A$ be on $l$ such that $\\angle A O B=60^{\\circ}$. Let $C$ be the foot of the perpendicular from $B$ to $O A$. Find the length of line segment $O C$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_11",
        "source_index": 1657,
        "difficulty": 3.5
      },
      {
        "id": "5c17d9c64ba345709359924e0c45014240fb3aec6fe40d3cf4d5fb8c072a6252",
        "domain": "omni",
        "question": "Call a positive integer $n$ weird if $n$ does not divide $(n-2)$!. Determine the number of weird numbers between 2 and 100 inclusive.",
        "answer": "26",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 817,
        "difficulty": 5.0
      },
      {
        "id": "53e89f2454523f5218c3268a9873b8ea5351507a8a91dcb8a6939bda9abad58d",
        "domain": "omni",
        "question": "Let $a$ be the proportion of teams that correctly answered problem 1 on the Guts round. Estimate $A=\\lfloor 10000a\\rfloor$. An estimate of $E$ earns $\\max (0,\\lfloor 20-|A-E| / 20\\rfloor)$ points. If you have forgotten, question 1 was the following: Two hexagons are attached to form a new polygon $P$. What is the minimum number of sides that $P$ can have?",
        "answer": "2539",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2647,
        "difficulty": 4.0
      },
      {
        "id": "1b39d663a2b0dad4925e1568dc3c74f3549e0b5ee7e99d4bd245b1f905d68d96",
        "domain": "omni",
        "question": "In a math test, there are easy and hard questions. The easy questions worth 3 points and the hard questions worth D points.\\\\\nIf all the questions begin to worth 4 points, the total punctuation of the test increases 16 points.\\\\\nInstead, if we exchange the questions scores, scoring D points for the easy questions and 3 for the hard ones, the total punctuation of the test is multiplied by $\\frac{3}{2}$.\\\\\nKnowing that the number of easy questions is 9 times bigger the number of hard questions, find the number of questions in this test.",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "lusophon_mathematical_olympiad",
        "source_index": 3890,
        "difficulty": 5.0
      },
      {
        "id": "624bbeaee965c5c49ba200267bf5c4cbc5b3c1a13f199168be130b855653228e",
        "domain": "omni",
        "question": "What is the largest positive integer $n$ that satisfies $n^{200}<3^{500}$?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2681,
        "difficulty": 2.0
      },
      {
        "id": "7841db48cf7b2ad1fb944ac32e39787df1076b69ce19f77f54bb3afdbbade05b",
        "domain": "omni",
        "question": "Four teams play in a tournament in which each team plays exactly one game against each of the other three teams. At the end of each game, either the two teams tie or one team wins and the other team loses. A team is awarded 3 points for a win, 0 points for a loss, and 1 point for a tie. If $S$ is the sum of the points of the four teams after the tournament is complete, which of the following values can $S$ not equal?",
        "answer": "11",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "fermat",
        "source_index": 3523,
        "difficulty": 2.5
      },
      {
        "id": "2869e23d7290c154de327f6ad4566329c33f1427681902cf0c9cd987014fa20c",
        "domain": "omni",
        "question": "Two circles have radii 13 and 30, and their centers are 41 units apart. The line through the centers of the two circles intersects the smaller circle at two points; let $A$ be the one outside the larger circle. Suppose $B$ is a point on the smaller circle and $C$ a point on the larger circle such that $B$ is the midpoint of $A C$. Compute the distance $A C$.",
        "answer": "12 \\sqrt{13}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 815,
        "difficulty": 5.25
      },
      {
        "id": "5ecac4778b7aa038be85776ea5b9ff49022ab08f3765515fc799ea2edfd5356d",
        "domain": "omni",
        "question": "Solve the system of equations: $20=4a^{2}+9b^{2}$ and $20+12ab=(2a+3b)^{2}$. Find $ab$.",
        "answer": "\\frac{20}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2411,
        "difficulty": 4.0
      },
      {
        "id": "fecaf84dcc6faa781a9756799c3a3c80da1c1c858fab8e1476bb5855deb51e96",
        "domain": "omni",
        "question": "The elevator buttons in Harvard's Science Center form a $3 \\times 2$ grid of identical buttons, and each button lights up when pressed. One day, a student is in the elevator when all the other lights in the elevator malfunction, so that only the buttons which are lit can be seen, but one cannot see which floors they correspond to. Given that at least one of the buttons is lit, how many distinct arrangements can the student observe? (For example, if only one button is lit, then the student will observe the same arrangement regardless of which button it is.)",
        "answer": "44",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2315,
        "difficulty": 4.0
      },
      {
        "id": "b15858278ab22cb88744da5cd0d914eaf7c786a05da93e053d1846fd41b0bc95",
        "domain": "omni",
        "question": "A monomial term $x_{i_{1}} x_{i_{2}} \\ldots x_{i_{k}}$ in the variables $x_{1}, x_{2}, \\ldots x_{8}$ is square-free if $i_{1}, i_{2}, \\ldots i_{k}$ are distinct. (A constant term such as 1 is considered square-free.) What is the sum of the coefficients of the squarefree terms in the following product? $$\\prod_{1 \\leq i<j \\leq 8}\\left(1+x_{i} x_{j}\\right)$$",
        "answer": "764",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 2354,
        "difficulty": 5.0
      },
      {
        "id": "a9c69d84912c40cb52921d1829cbd9e2ce46f27611b8e3f48f95115b37d733ed",
        "domain": "omni",
        "question": "Let $P$ be a regular $2006$-gon. A diagonal is called [i]good[/i] if its endpoints divide the boundary of $P$ into two parts, each composed of an odd number of sides of $P$. The sides of $P$ are also called [i]good[/i].\nSuppose $P$ has been dissected into triangles by $2003$ diagonals, no two of which have a common point in the interior of $P$. Find the maximum number of isosceles triangles having two good sides that could appear in such a configuration.",
        "answer": "1003",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "imo",
        "source_index": 3929,
        "difficulty": 8.0
      },
      {
        "id": "51c0a47abc491e3c6248890d6ac41ddb41c00bdc03e077ef5b1822556982b940",
        "domain": "omni",
        "question": "A peacock is a ten-digit positive integer that uses each digit exactly once. Compute the number of peacocks that are exactly twice another peacock.",
        "answer": "184320",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1116,
        "difficulty": 6.0
      },
      {
        "id": "afbe05d4c359e6d4744b2a3c73a4a59a2ec34a694fc08e2f6b53c97b9a590e9b",
        "domain": "omni",
        "question": "Determine the value of the sum $$\\frac{3}{1^{2} \\cdot 2^{2}}+\\frac{5}{2^{2} \\cdot 3^{2}}+\\frac{7}{3^{2} \\cdot 4^{2}}+\\cdots+\\frac{29}{14^{2} \\cdot 15^{2}}$$",
        "answer": "\\frac{224}{225}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1347,
        "difficulty": 5.0
      },
      {
        "id": "feedff347d070efaff8db6c18a3951d74ea43adbec625f1f5bc8c26ade3fd434",
        "domain": "omni",
        "question": "Find the total number of different integer values the function $$f(x)=[x]+[2 x]+\\left[\\frac{5 x}{3}\\right]+[3 x]+[4 x]$$ takes for real numbers $x$ with $0 \\leq x \\leq 100$. Note: $[t]$ is the largest integer that does not exceed $t$.",
        "answer": "734",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "apmoapmo_sol",
        "source_index": 1570,
        "difficulty": 6.0
      },
      {
        "id": "ef4c1c8242d9d7271bf46cccf46c67b8b1ef0413d3e11f8c1db2d0b135ed600f",
        "domain": "omni",
        "question": "For $a$ a positive real number, let $x_{1}, x_{2}, x_{3}$ be the roots of the equation $x^{3}-a x^{2}+a x-a=0$. Determine the smallest possible value of $x_{1}^{3}+x_{2}^{3}+x_{3}^{3}-3 x_{1} x_{2} x_{3}$.",
        "answer": "-4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1531,
        "difficulty": 4.5
      },
      {
        "id": "69aed55faa3a9f551290457827bf23dfd70542771ea0dea9367e3c9c143b3fcf",
        "domain": "omni",
        "question": "Let $n$ be a fixed positive integer. Determine the smallest possible rank of an $n \\times n$ matrix that has zeros along the main diagonal and strictly positive real numbers off the main diagonal.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Matrices",
        "source": "imc",
        "source_index": 2640,
        "difficulty": 7.0
      },
      {
        "id": "290c0f9553e55a508dd293405c9d0c0a529346bc1d60f1a9c6613eccbcb63095",
        "domain": "omni",
        "question": "In equilateral triangle $ABC$ with side length 2, let the parabola with focus $A$ and directrix $BC$ intersect sides $AB$ and $AC$ at $A_{1}$ and $A_{2}$, respectively. Similarly, let the parabola with focus $B$ and directrix $CA$ intersect sides $BC$ and $BA$ at $B_{1}$ and $B_{2}$, respectively. Finally, let the parabola with focus $C$ and directrix $AB$ intersect sides $CA$ and $CB$ at $C_{1}$ and $C_{2}$, respectively. Find the perimeter of the triangle formed by lines $A_{1}A_{2}, B_{1}B_{2}, C_{1}C_{2}$.",
        "answer": "66-36\\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2226,
        "difficulty": 5.0
      },
      {
        "id": "2c7b9fee09d0226908599597f956e1a69d5a7c5c4e97c64c5c38d9ffd1567534",
        "domain": "omni",
        "question": "Points $X$ and $Y$ are inside a unit square. The score of a vertex of the square is the minimum distance from that vertex to $X$ or $Y$. What is the minimum possible sum of the scores of the vertices of the square?",
        "answer": "\\frac{\\sqrt{6}+\\sqrt{2}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 780,
        "difficulty": 4.0
      },
      {
        "id": "91872f83481daf64201351930d2052dbee645cb60c4715bb6a7e24898099c28c",
        "domain": "omni",
        "question": "Over all real numbers $x$ and $y$ such that $$x^{3}=3 x+y \\quad \\text { and } \\quad y^{3}=3 y+x$$ compute the sum of all possible values of $x^{2}+y^{2}$.",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1818,
        "difficulty": 5.0
      },
      {
        "id": "5266672dd64ea9e571a5c2533dad07cfc51c9be0eb1d541360cce428fca518cf",
        "domain": "omni",
        "question": "How many functions $f:\\{0,1\\}^{3} \\rightarrow\\{0,1\\}$ satisfy the property that, for all ordered triples \\left(a_{1}, a_{2}, a_{3}\\right) and \\left(b_{1}, b_{2}, b_{3}\\right) such that $a_{i} \\geq b_{i}$ for all $i, f\\left(a_{1}, a_{2}, a_{3}\\right) \\geq f\\left(b_{1}, b_{2}, b_{3}\\right)$?",
        "answer": "20",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_11",
        "source_index": 1846,
        "difficulty": 5.0
      },
      {
        "id": "7b78e4cd754e6295f1e3d6931a5a527a7a523c1532d419f5dc7a84b7cacf91a5",
        "domain": "omni",
        "question": "For how many integer values of $b$ does there exist a polynomial function with integer coefficients such that $f(2)=2010$ and $f(b)=8$?",
        "answer": "32",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 1691,
        "difficulty": 5.0
      },
      {
        "id": "eba09940b4e9edd03febef7d5a53649d0b64e15bac34f73f88374f964f173d5b",
        "domain": "omni",
        "question": "Violet has one-half of the money she needs to buy her mother a necklace. After her sister gives her $\\$30$, she has three-quarters of the amount she needs. How much will Violet's father give her?",
        "answer": "$30",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3417,
        "difficulty": 1.5
      },
      {
        "id": "4649f15e082f4970e9d8862d93f3444976819d7e13ad211b00d2dc230a01b696",
        "domain": "omni",
        "question": "If $x+\\sqrt{25}=\\sqrt{36}$, what is the value of $x$?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2703,
        "difficulty": 1.0
      },
      {
        "id": "4289f11d9b18f91dc28f47aee65da53aa725900681db5365177823170edd25b8",
        "domain": "omni",
        "question": "If \\( N \\) is the smallest positive integer whose digits have a product of 2700, what is the sum of the digits of \\( N \\)?",
        "answer": "27",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2717,
        "difficulty": 2.0
      },
      {
        "id": "6023c9d729d622ea2718b10070eaad18a503f78fb94c63eabdc96327b7c1fa56",
        "domain": "omni",
        "question": "Consider the set \\(S\\) of all complex numbers \\(z\\) with nonnegative real and imaginary part such that \\(\\left|z^{2}+2\\right| \\leq|z|\\). Across all \\(z \\in S\\), compute the minimum possible value of \\(\\tan \\theta\\), where \\(\\theta\\) is the angle formed between \\(z\\) and the real axis.",
        "answer": "\\sqrt{7}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2376,
        "difficulty": 4.5
      },
      {
        "id": "8b820ba98d5317bc56796c8c1b2961e3ffaef59a2a753b22d8df894d5c804e2b",
        "domain": "omni",
        "question": "Originally, every square of $8 \\times 8$ chessboard contains a rook. One by one, rooks which attack an odd number of others are removed. Find the maximal number of rooks that can be removed. (A rook attacks another rook if they are on the same row or column and there are no other rooks between them.)",
        "answer": "59",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "ToT",
        "source_index": 3863,
        "difficulty": 6.0
      },
      {
        "id": "ca17e03a1bc68cf1eacde5b49925f36dd135ea2f8c12a80fc785e64b27a57fd9",
        "domain": "omni",
        "question": "In the list 7, 9, 10, 11, 18, which number is the average (mean) of the other four numbers?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3480,
        "difficulty": 1.0
      },
      {
        "id": "74cedb27d0db2f4984232108c2bd596c838d01a8dd32047e9364296bb0233d83",
        "domain": "omni",
        "question": "$A B C D$ is a cyclic quadrilateral in which $A B=4, B C=3, C D=2$, and $A D=5$. Diagonals $A C$ and $B D$ intersect at $X$. A circle $\\omega$ passes through $A$ and is tangent to $B D$ at $X . \\omega$ intersects $A B$ and $A D$ at $Y$ and $Z$ respectively. Compute $Y Z / B D$.",
        "answer": "\\frac{115}{143}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1528,
        "difficulty": 5.0
      },
      {
        "id": "9710e5ba0efe28b2f2ec3f40d4bdbccc2366f81cd0c05a2f4ecc029a0c8775a1",
        "domain": "omni",
        "question": "Determine the number of quadratic polynomials $P(x)=p_{1} x^{2}+p_{2} x-p_{3}$, where $p_{1}, p_{2}, p_{3}$ are not necessarily distinct (positive) prime numbers less than 50, whose roots are distinct rational numbers.",
        "answer": "31",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2132,
        "difficulty": 4.5
      },
      {
        "id": "798798cede3f5c5bb1f2188d5ae003467b8a8877fe0707dae191872082117575",
        "domain": "omni",
        "question": "$A B C D$ is a parallelogram satisfying $A B=7, B C=2$, and $\\angle D A B=120^{\\circ}$. Parallelogram $E C F A$ is contained in $A B C D$ and is similar to it. Find the ratio of the area of $E C F A$ to the area of $A B C D$.",
        "answer": "\\frac{39}{67}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2208,
        "difficulty": 4.0
      },
      {
        "id": "438e665f56e03f775d843296d1c7f8401e3f020e5fa3133de386b0bbe0a1282c",
        "domain": "omni",
        "question": "Let $ABC$ be a triangle in the plane with $AB=13, BC=14, AC=15$. Let $M_{n}$ denote the smallest possible value of $\\left(AP^{n}+BP^{n}+CP^{n}\\right)^{\\frac{1}{n}}$ over all points $P$ in the plane. Find $\\lim _{n \\rightarrow \\infty} M_{n}$.",
        "answer": "\\frac{65}{8}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 448,
        "difficulty": 5.0
      },
      {
        "id": "03a1a6f05c70161049724cf9c9d9be92b2b72bedea80602eb28c9ca2ab893328",
        "domain": "omni",
        "question": "Let \\(p\\) be the answer to this question. If a point is chosen uniformly at random from the square bounded by \\(x=0, x=1, y=0\\), and \\(y=1\\), what is the probability that at least one of its coordinates is greater than \\(p\\)?",
        "answer": "\\frac{\\sqrt{5}-1}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 357,
        "difficulty": 5.0
      },
      {
        "id": "841ec6b5a085b1ac431f5a7dc4ae41034e1a596aa54f8c6bae95784078865851",
        "domain": "omni",
        "question": "Find the smallest positive integer  $n$ such that if $n$ squares of a $1000 \\times 1000$ chessboard are colored, then there will exist three colored squares whose centers form a right triangle with sides parallel to the edges of the board.",
        "answer": "\\boxed{1999}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usamo",
        "source_index": 238,
        "difficulty": 7.0
      },
      {
        "id": "33e28abbd0dc1dcf9eb1e896e724b231ce4be14334c46f9a8bf3070f0658b1fa",
        "domain": "omni",
        "question": "A certain cafeteria serves ham and cheese sandwiches, ham and tomato sandwiches, and tomato and cheese sandwiches. It is common for one meal to include multiple types of sandwiches. On a certain day, it was found that 80 customers had meals which contained both ham and cheese; 90 had meals containing both ham and tomatoes; 100 had meals containing both tomatoes and cheese. 20 customers' meals included all three ingredients. How many customers were there?",
        "answer": "230",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1095,
        "difficulty": 4.0
      },
      {
        "id": "609b081c9b1a03f407fd880d213dc0213e721c2dcd2289c3a0e3ec26db76c191",
        "domain": "omni",
        "question": "Let $ABC$ be a right triangle with hypotenuse $AC$. Let $B^{\\prime}$ be the reflection of point $B$ across $AC$, and let $C^{\\prime}$ be the reflection of $C$ across $AB^{\\prime}$. Find the ratio of $[BCB^{\\prime}]$ to $[BC^{\\prime}B^{\\prime}]$.",
        "answer": "1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1704,
        "difficulty": 5.0
      },
      {
        "id": "c4acdc50a577a1d8c0de4ece6c7409de4ea2398280391ff28aaea8ec6180b2c2",
        "domain": "omni",
        "question": "Farmer James wishes to cover a circle with circumference $10 \\pi$ with six different types of colored arcs. Each type of arc has radius 5, has length either $\\pi$ or $2 \\pi$, and is colored either red, green, or blue. He has an unlimited number of each of the six arc types. He wishes to completely cover his circle without overlap, subject to the following conditions: Any two adjacent arcs are of different colors. Any three adjacent arcs where the middle arc has length $\\pi$ are of three different colors. Find the number of distinct ways Farmer James can cover his circle. Here, two coverings are equivalent if and only if they are rotations of one another. In particular, two colorings are considered distinct if they are reflections of one another, but not rotations of one another.",
        "answer": "93",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1198,
        "difficulty": 6.0
      },
      {
        "id": "aa7fbaf617866e21843260585940537924f239b2606a9cdeed3502b881cab03e",
        "domain": "omni",
        "question": "In $\\triangle A B C$, the incircle centered at $I$ touches sides $A B$ and $B C$ at $X$ and $Y$, respectively. Additionally, the area of quadrilateral $B X I Y$ is $\\frac{2}{5}$ of the area of $A B C$. Let $p$ be the smallest possible perimeter of a $\\triangle A B C$ that meets these conditions and has integer side lengths. Find the smallest possible area of such a triangle with perimeter $p$.",
        "answer": "2 \\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1970,
        "difficulty": 4.0
      },
      {
        "id": "17a4162dd1ce9f161eefd4954226a90e29270eb704190c5b5c7b5ed70b6d67f8",
        "domain": "omni",
        "question": "Which number is closest to \\(-3.4\\) on a number line?",
        "answer": "-3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3476,
        "difficulty": 1.0
      },
      {
        "id": "0d2ccf223f590cabfbfd33cb9327c2b35746753781aec4c636dc8aec598d0788",
        "domain": "omni",
        "question": "Three fair six-sided dice, each numbered 1 through 6 , are rolled. What is the probability that the three numbers that come up can form the sides of a triangle?",
        "answer": "37/72",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1027,
        "difficulty": 4.0
      },
      {
        "id": "dce8acd2cf5f5d6c77d9807ba229c1918be37c08e442013750dc9763558d3857",
        "domain": "omni",
        "question": "If $x + 2y = 30$, what is the value of $\\frac{x}{5} + \\frac{2y}{3} + \\frac{2y}{5} + \\frac{x}{3}$?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2750,
        "difficulty": 2.5
      },
      {
        "id": "3b3a61e1faf11ce69e7b812d2f89c4ed1e61294c49981332986dfa983bedd299",
        "domain": "omni",
        "question": "There are 30 people in a room, 60\\% of whom are men. If no men enter or leave the room, how many women must enter the room so that 40\\% of the total number of people in the room are men?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2725,
        "difficulty": 2.0
      },
      {
        "id": "6d48fba34dfb10f879c021ccc8d6bd0cba6bff058b0b64271645f2aeca32014a",
        "domain": "omni",
        "question": "Kimothy starts in the bottom-left square of a 4 by 4 chessboard. In one step, he can move up, down, left, or right to an adjacent square. Kimothy takes 16 steps and ends up where he started, visiting each square exactly once (except for his starting/ending square). How many paths could he have taken?",
        "answer": "12",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2388,
        "difficulty": 4.5
      },
      {
        "id": "9b6dc76707e8352ac61d918b4fc0a625996dd692eab16e285a7a0348ecfa1061",
        "domain": "omni",
        "question": "Consider a $9 \\times 9$ grid of squares. Haruki fills each square in this grid with an integer between 1 and 9 , inclusive. The grid is called a super-sudoku if each of the following three conditions hold: - Each column in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. - Each row in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. - Each $3 \\times 3$ subsquare in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. How many possible super-sudoku grids are there?",
        "answer": "0",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2260,
        "difficulty": 4.0
      },
      {
        "id": "3c42d230a52d1e3b6cb449ee24f05ebda270a628a8c4acc8076e5b20d3d818ad",
        "domain": "omni",
        "question": "Number $a$ is such that $\\forall a_1, a_2, a_3, a_4 \\in \\mathbb{R}$, there are integers $k_1, k_2, k_3, k_4$ such that $\\sum_{1 \\leq i < j \\leq 4} ((a_i - k_i) - (a_j - k_j))^2 \\leq a$. Find the minimum of $a$.",
        "answer": "1.25",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "china_team_selection_test",
        "source_index": 102,
        "difficulty": 9.0
      },
      {
        "id": "7fccddc772a637ed3598782bab1ce3162678d5b695623a446fc6b16e3361abe9",
        "domain": "omni",
        "question": "For how many pairs of sequences of nonnegative integers $\\left(b_{1}, b_{2}, \\ldots, b_{2018}\\right)$ and $\\left(c_{1}, c_{2}, \\ldots, c_{2018}\\right)$ does there exist a sequence of nonnegative integers $\\left(a_{0}, \\ldots, a_{2018}\\right)$ with the following properties: For $0 \\leq i \\leq 2018, a_{i}<2^{2018}$; For $1 \\leq i \\leq 2018, b_{i}=a_{i-1}+a_{i}$ and $c_{i}=a_{i-1} \\mid a_{i}$ where $\\mid$ denotes the bitwise or operation?",
        "answer": "\\left(2^{2019}-1\\right)^{2018}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1593,
        "difficulty": 6.5
      },
      {
        "id": "c844250a4d75e5e067c63221346d9eb3017d0dd666acf325c04bcac021c3d67b",
        "domain": "omni",
        "question": "The number $$316990099009901=\\frac{32016000000000001}{101}$$ is the product of two distinct prime numbers. Compute the smaller of these two primes.",
        "answer": "4002001",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1493,
        "difficulty": 5.0
      },
      {
        "id": "063cee3f53bde00a8c1359a22f37848d4af484ccbc414b00d9eb37267bde7404",
        "domain": "omni",
        "question": "Mattis is hosting a badminton tournament for $40$ players on $20$ courts numbered from $1$ to $20$. The players are distributed with $2$ players on each court. In each round a winner is determined on each court. Afterwards, the player who lost on court $1$, and the player who won on court $20$ stay in place. For the remaining $38$ players, the winner on court $i$ moves to court $i + 1$ and the loser moves to court $i - 1$. The tournament continues until every player has played every other player at least once. What is the minimal number of rounds the tournament can last?",
        "answer": "39",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "baltic_way",
        "source_index": 3613,
        "difficulty": 5.5
      },
      {
        "id": "5a62db95ea6d771bc57995b1a04b962f0b6ddefb8172353e541cf1b8fc7646d8",
        "domain": "omni",
        "question": "How many words are there in a language that are 10 letters long and begin with a vowel, given that the language uses only the letters A, B, C, D, and E, where A and E are vowels, and B, C, and D are consonants, and a word does not include the same letter twice in a row or two vowels in a row?",
        "answer": "199776",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 3106,
        "difficulty": 2.5
      },
      {
        "id": "7f86e9ee617d655e4bd22f4e5181346f4df786da00654fac0070ed9cfa2506ce",
        "domain": "omni",
        "question": "Find the largest integer $n$ such that the following holds: there exists a set of $n$ points in the plane such that, for any choice of three of them, some two are unit distance apart.",
        "answer": "7",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_11",
        "source_index": 1766,
        "difficulty": 5.0
      },
      {
        "id": "95a82b55b43b564445f0cdbb213e82940098d4f5e16e4d8b7e9e3d7a790ca773",
        "domain": "omni",
        "question": "What is the smallest integer $n$ , greater than one, for which the root-mean-square of the first $n$ positive integers is an integer?\n$\\mathbf{Note.}$ The root-mean-square of $n$ numbers $a_1, a_2, \\cdots, a_n$ is defined to be \\[\\left[\\frac{a_1^2 + a_2^2 + \\cdots + a_n^2}n\\right]^{1/2}\\]",
        "answer": "\\(\\boxed{337}\\)",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions; Mathematics -> Number Theory -> Other",
        "source": "usamo",
        "source_index": 131,
        "difficulty": 7.0
      },
      {
        "id": "0ae0cbc3eb44ed7802c9d0700cfae8357e1b449984381504123d00cfc2010147",
        "domain": "omni",
        "question": "What number is placed in the shaded circle if each of the numbers $1,5,6,7,13,14,17,22,26$ is placed in a different circle, the numbers 13 and 17 are placed as shown, and Jen calculates the average of the numbers in the first three circles, the average of the numbers in the middle three circles, and the average of the numbers in the last three circles, and these three averages are equal?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2964,
        "difficulty": 2.0
      },
      {
        "id": "9599f5595a295b7edf23aa491a7198dc72256210efc0543b75ab5e70dbfd6a61",
        "domain": "omni",
        "question": "Six consecutive integers are written on a blackboard. When one of them is erased, the sum of the remaining five integers is 2012. What is the sum of the digits of the integer that was erased?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2674,
        "difficulty": 2.0
      },
      {
        "id": "798c81e6b16142bd605f8de2a98ae755130380330ea0b8d0f933a661c71169b0",
        "domain": "omni",
        "question": "All subscripts in this problem are to be considered modulo 6 , that means for example that $\\omega_{7}$ is the same as $\\omega_{1}$. Let $\\omega_{1}, \\ldots \\omega_{6}$ be circles of radius $r$, whose centers lie on a regular hexagon of side length 1 . Let $P_{i}$ be the intersection of $\\omega_{i}$ and $\\omega_{i+1}$ that lies further from the center of the hexagon, for $i=1, \\ldots 6$. Let $Q_{i}, i=1 \\ldots 6$, lie on $\\omega_{i}$ such that $Q_{i}, P_{i}, Q_{i+1}$ are colinear. Find the number of possible values of $r$.",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 1109,
        "difficulty": 5.0
      },
      {
        "id": "034195347d74b3132faf2152ead7e252503ab38e2281de43d991c2a38f96a16d",
        "domain": "omni",
        "question": "Consider the sequence: $x_1=19,x_2=95,x_{n+2}=\\text{lcm} (x_{n+1},x_n)+x_n$, for $n>1$, where $\\text{lcm} (a,b)$ means the least common multiple of $a$ and $b$. Find the greatest common divisor of $x_{1995}$ and $x_{1996}$.",
        "answer": "19",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Number Theory -> Factorization",
        "source": "baltic_way",
        "source_index": 3645,
        "difficulty": 4.5
      },
      {
        "id": "4205d75d45cc57f887bc3524a26e6dc65e1eef5ee9a644864589d57e38c769d0",
        "domain": "omni",
        "question": "Admiral Ackbar needs to send a 5-character message through hyperspace to the Rebels. Each character is a lowercase letter, and the same letter may appear more than once in a message. When the message is beamed through hyperspace, the characters come out in a random order. Ackbar chooses his message so that the Rebels have at least a $\\frac{1}{2}$ chance of getting the same message he sent. How many distinct messages could he send?",
        "answer": "26",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1781,
        "difficulty": 3.5
      },
      {
        "id": "142269e08e66a24479a40f50806703ac67f6cb0fa8a2a344ead0993fe413c4be",
        "domain": "omni",
        "question": "Compute the number of ordered quintuples of nonnegative integers $(a_{1}, a_{2}, a_{3}, a_{4}, a_{5})$ such that $0 \\leq a_{1}, a_{2}, a_{3}, a_{4}, a_{5} \\leq 7$ and 5 divides $2^{a_{1}}+2^{a_{2}}+2^{a_{3}}+2^{a_{4}}+2^{a_{5}}$.",
        "answer": "6528",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Modular Arithmetic -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 908,
        "difficulty": 5.25
      },
      {
        "id": "41bb9cd4140ad210a8d49c30b2c5d40f23d9041fc679ddcb59e3e0367f22359c",
        "domain": "omni",
        "question": "Let $A B C$ be an acute isosceles triangle with orthocenter $H$. Let $M$ and $N$ be the midpoints of sides $\\overline{A B}$ and $\\overline{A C}$, respectively. The circumcircle of triangle $M H N$ intersects line $B C$ at two points $X$ and $Y$. Given $X Y=A B=A C=2$, compute $B C^{2}$.",
        "answer": "2(\\sqrt{17}-1)",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 863,
        "difficulty": 5.0
      },
      {
        "id": "a80ec0179dd4cb32fff2eb13e02f556ab361ea13c2fc42e7b31df16f77abdb25",
        "domain": "omni",
        "question": "In the game of projective set, each card contains some nonempty subset of six distinguishable dots. A projective set deck consists of one card for each of the 63 possible nonempty subsets of dots. How many collections of five cards have an even number of each dot? The order in which the cards appear does not matter.",
        "answer": "109368",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2451,
        "difficulty": 5.0
      },
      {
        "id": "900173d13601587f42464064a42d68e11130dae4769d780a236464cc856d40f0",
        "domain": "omni",
        "question": "Points $E, F, G, H$ are chosen on segments $A B, B C, C D, D A$, respectively, of square $A B C D$. Given that segment $E G$ has length 7 , segment $F H$ has length 8 , and that $E G$ and $F H$ intersect inside $A B C D$ at an acute angle of $30^{\\circ}$, then compute the area of square $A B C D$.",
        "answer": "\\frac{784}{19}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 1914,
        "difficulty": 5.0
      },
      {
        "id": "b61856418a5ae48822885384a359f9a4ab47d68e22dd73cdcdd69fc52923dd1d",
        "domain": "omni",
        "question": "Given a positive integer $k$, let \\|k\\| denote the absolute difference between $k$ and the nearest perfect square. For example, \\|13\\|=3 since the nearest perfect square to 13 is 16. Compute the smallest positive integer $n$ such that $\\frac{\\|1\\|+\\|2\\|+\\cdots+\\|n\\|}{n}=100$.",
        "answer": "89800",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1518,
        "difficulty": 5.0
      },
      {
        "id": "2ddbb7de1fcfd464f85603c708aa3592526b677970f3b1b25122706df8d57cb1",
        "domain": "omni",
        "question": "Suppose $x^{3}-a x^{2}+b x-48$ is a polynomial with three positive roots $p, q$, and $r$ such that $p<q<r$. What is the minimum possible value of $1 / p+2 / q+3 / r$ ?",
        "answer": "3 / 2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 991,
        "difficulty": 5.0
      },
      {
        "id": "100ea40c9aed1c6dfa61fa2aebaf72b3e19b75797a571ab3d342ff5ce305e03a",
        "domain": "omni",
        "question": "Compute \\( \\frac{2^{3}-1}{2^{3}+1} \\cdot \\frac{3^{3}-1}{3^{3}+1} \\cdot \\frac{4^{3}-1}{4^{3}+1} \\cdot \\frac{5^{3}-1}{5^{3}+1} \\cdot \\frac{6^{3}-1}{6^{3}+1} \\).",
        "answer": "43/63",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1386,
        "difficulty": 5.0
      },
      {
        "id": "ade1d167875591bc154b68859a06bcc174d0f5ba03c73ceafb5d87959454553b",
        "domain": "omni",
        "question": "When 542 is multiplied by 3, what is the ones (units) digit of the result?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3073,
        "difficulty": 1.0
      },
      {
        "id": "6c2834132d61c6f7ff7242949fa5e06f73d98d2188182713ad3ba1d2760025ad",
        "domain": "omni",
        "question": "Two 18-24-30 triangles in the plane share the same circumcircle as well as the same incircle. What's the area of the region common to both the triangles?",
        "answer": "132",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 730,
        "difficulty": 5.0
      },
      {
        "id": "1337bb78d940de6621c029dea1f6b9ca2004c2fad42ef207ce725fbb29dc1e40",
        "domain": "omni",
        "question": "Allen and Yang want to share the numbers \\(1,2,3,4,5,6,7,8,9,10\\). How many ways are there to split all ten numbers among Allen and Yang so that each person gets at least one number, and either Allen's numbers or Yang's numbers sum to an even number?",
        "answer": "1022",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1569,
        "difficulty": 4.5
      },
      {
        "id": "ff8a8064a3d557fbfb736632ea1cf48c48da50c2eb4cc54e86c817b8da6f8616",
        "domain": "omni",
        "question": "Kevin starts with the vectors \\((1,0)\\) and \\((0,1)\\) and at each time step, he replaces one of the vectors with their sum. Find the cotangent of the minimum possible angle between the vectors after 8 time steps.",
        "answer": "987",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Linear Algebra -> Vectors",
        "source": "HMMT_2",
        "source_index": 1552,
        "difficulty": 5.0
      },
      {
        "id": "bb70154e2acfd67b665a5cd5e791e0219f92476cf9f67a11f299bffa5cc39663",
        "domain": "omni",
        "question": "A tourist is learning an incorrect way to sort a permutation $(p_{1}, \\ldots, p_{n})$ of the integers $(1, \\ldots, n)$. We define a fix on two adjacent elements $p_{i}$ and $p_{i+1}$, to be an operation which swaps the two elements if $p_{i}>p_{i+1}$, and does nothing otherwise. The tourist performs $n-1$ rounds of fixes, numbered $a=1,2, \\ldots, n-1$. In round $a$ of fixes, the tourist fixes $p_{a}$ and $p_{a+1}$, then $p_{a+1}$ and $p_{a+2}$, and so on, up to $p_{n-1}$ and $p_{n}$. In this process, there are $(n-1)+(n-2)+\\cdots+1=\\frac{n(n-1)}{2}$ total fixes performed. How many permutations of $(1, \\ldots, 2018)$ can the tourist start with to obtain $(1, \\ldots, 2018)$ after performing these steps?",
        "answer": "1009! \\cdot 1010!",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1110,
        "difficulty": 5.5
      },
      {
        "id": "216140b4c6b4bc18e5123a974bdc408a06b9fa84514da0a2a34c093049761f24",
        "domain": "omni",
        "question": "There are 101 people participating in a Secret Santa gift exchange. As usual each person is randomly assigned another person for whom (s)he has to get a gift, such that each person gives and receives exactly one gift and no one gives a gift to themself. What is the probability that the first person neither gives gifts to or receives gifts from the second or third person? Express your answer as a decimal rounded to five decimal places.",
        "answer": "0.96039",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 2394,
        "difficulty": 5.25
      },
      {
        "id": "6554973e5c93934e7e9ae589e2d1c45b0de7244fd4bd753bdd967199a1747d20",
        "domain": "omni",
        "question": "What is the average (mean) number of hamburgers eaten per student if 12 students ate 0 hamburgers, 14 students ate 1 hamburger, 8 students ate 2 hamburgers, 4 students ate 3 hamburgers, and 2 students ate 4 hamburgers?",
        "answer": "1.25",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3164,
        "difficulty": 1.5
      },
      {
        "id": "98db58602facd72a50f7a45bc0398369f84a01f3bf9dacb05a3b98cc5185a55e",
        "domain": "omni",
        "question": "Find the sum of the reciprocals of all the (positive) divisors of 144.",
        "answer": "403/144",
        "topic": "Mathematics -> Number Theory -> Divisors -> Other",
        "source": "HMMT_2",
        "source_index": 507,
        "difficulty": 3.5
      },
      {
        "id": "2d992e9e715ce7eff026c4f87ba931a14fd1f39575b2dd8e6dc906667c8502d6",
        "domain": "omni",
        "question": "Today, Ivan the Confessor prefers continuous functions $f:[0,1]\\to\\mathbb{R}$ satisfying $f(x)+f(y)\\geq |x-y|$ for all pairs $x,y\\in [0,1]$. Find the minimum of $\\int_0^1 f$ over all preferred functions.\n\n(",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "imc",
        "source_index": 3907,
        "difficulty": 8.0
      },
      {
        "id": "35ae7300397f617ae18e3288b32a47de60219119631f78e64f6e8340fd4f747b",
        "domain": "omni",
        "question": "Suppose $A B C$ is a triangle with circumcenter $O$ and orthocenter $H$ such that $A, B, C, O$, and $H$ are all on distinct points with integer coordinates. What is the second smallest possible value of the circumradius of $A B C$ ?",
        "answer": "\\sqrt{10}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 734,
        "difficulty": 5.0
      },
      {
        "id": "ad7eab32a9988d73a34a48b279bdf0daf7d86e29c5ff409cb43c32ebe1ee54b3",
        "domain": "omni",
        "question": "You are standing at a pole and a snail is moving directly away from the pole at $1 \\mathrm{~cm} / \\mathrm{s}$. When the snail is 1 meter away, you start 'Round 1'. In Round $n(n \\geq 1)$, you move directly toward the snail at $n+1 \\mathrm{~cm} / \\mathrm{s}$. When you reach the snail, you immediately turn around and move back to the starting pole at $n+1 \\mathrm{~cm} / \\mathrm{s}$. When you reach the pole, you immediately turn around and Round $n+1$ begins. At the start of Round 100, how many meters away is the snail?",
        "answer": "5050",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2111,
        "difficulty": 4.0
      },
      {
        "id": "c936216e51a6b5271388915a934afaa0d5288e669f4b60e7f92afbcc7eaac6e6",
        "domain": "omni",
        "question": "Let $n$ be the smallest positive integer with exactly 2015 positive factors. What is the sum of the (not necessarily distinct) prime factors of $n$?",
        "answer": "116",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1746,
        "difficulty": 5.0
      },
      {
        "id": "18e23694415cf86cebe08e0e2da4c1973def5a129c7e3514a1a8eef50c1a9300",
        "domain": "omni",
        "question": "Quadrilateral $ABCD$ has $\\angle BCD=\\angle DAB=90^{\\circ}$. The perimeter of $ABCD$ is 224 and its area is 2205. One side of $ABCD$ has length 7. The remaining three sides have integer lengths. What is the integer formed by the rightmost two digits of the sum of the squares of the side lengths of $ABCD$?",
        "answer": "60",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2868,
        "difficulty": 2.5
      },
      {
        "id": "840c5559accc612ba06fb78d2a77bb53d1ed9499066c3bc42a6ec95c718eb1a4",
        "domain": "omni",
        "question": "For an integer $n$, let $f(n)$ denote the number of pairs $(x, y)$ of integers such that $x^{2}+x y+y^{2}=n$. Compute the sum $\\sum_{n=1}^{10^{6}} n f(n)$",
        "answer": "1.813759629294 \\cdot 10^{12}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Multi-variable",
        "source": "HMMT_2",
        "source_index": 930,
        "difficulty": 5.0
      },
      {
        "id": "749f7342575fad18e696eb11335a5908c97853bff240c9283fbd715a64ca725a",
        "domain": "omni",
        "question": "One integer is selected at random from the following list of 15 integers: $1,2,2,3,3,3,4,4,4,4,5,5,5,5,5$. The probability that the selected integer is equal to $n$ is $\\frac{1}{3}$. What is the value of $n$?",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "fermat",
        "source_index": 2789,
        "difficulty": 2.0
      },
      {
        "id": "fcece29e184778629d0ac1cf58d1a4b3de17fd42095534fcf61a610123420cf7",
        "domain": "omni",
        "question": "Let $\\omega_{1}$ and $\\omega_{2}$ be two non-intersecting circles. Suppose the following three conditions hold: - The length of a common internal tangent of $\\omega_{1}$ and $\\omega_{2}$ is equal to 19 . - The length of a common external tangent of $\\omega_{1}$ and $\\omega_{2}$ is equal to 37 . - If two points $X$ and $Y$ are selected on $\\omega_{1}$ and $\\omega_{2}$, respectively, uniformly at random, then the expected value of $X Y^{2}$ is 2023 . Compute the distance between the centers of $\\omega_{1}$ and $\\omega_{2}$.",
        "answer": "38",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 2015,
        "difficulty": 5.0
      },
      {
        "id": "85d9e527ef8678fea10fe5ba14cabef83558a458961cdb2abbc06bf58b1aec78",
        "domain": "omni",
        "question": "Find the maximum possible value of $H \\cdot M \\cdot M \\cdot T$ over all ordered triples $(H, M, T)$ of integers such that $H \\cdot M \\cdot M \\cdot T=H+M+M+T$.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 621,
        "difficulty": 5.0
      },
      {
        "id": "ee95cd61c1d00d19a154f67e45671f019530061a9a56980106306f081f730765",
        "domain": "omni",
        "question": "Denote $\\phi=\\frac{1+\\sqrt{5}}{2}$ and consider the set of all finite binary strings without leading zeroes. Each string $S$ has a \"base-$\\phi$ \" value $p(S)$. For example, $p(1101)=\\phi^{3}+\\phi^{2}+1$. For any positive integer $n$, let $f(n)$ be the number of such strings $S$ that satisfy $p(S)=\\frac{\\phi^{48 n}-1}{\\phi^{48}-1}$. The sequence of fractions $\\frac{f(n+1)}{f(n)}$ approaches a real number $c$ as $n$ goes to infinity. Determine the value of $c$.",
        "answer": "\\frac{25+3 \\sqrt{69}}{2}",
        "topic": "",
        "source": "HMMT_11",
        "source_index": 2066,
        "difficulty": 5.0
      },
      {
        "id": "8b6208decbec0a3656de518d4265972c7d4eeaaec7dcf4eef3268704804a73f9",
        "domain": "omni",
        "question": "A rectangular pool table has vertices at $(0,0)(12,0)(0,10)$, and $(12,10)$. There are pockets only in the four corners. A ball is hit from $(0,0)$ along the line $y=x$ and bounces off several walls before eventually entering a pocket. Find the number of walls that the ball bounces off of before entering a pocket.",
        "answer": "9",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2425,
        "difficulty": 4.5
      },
      {
        "id": "67680ff470b67487da83b730b940a2ae6e735b568e46e9af8127d4363acdf69e",
        "domain": "omni",
        "question": "The line with equation $y = x$ is translated 3 units to the right and 2 units down. What is the $y$-intercept of the resulting line?",
        "answer": "-5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "fermat",
        "source_index": 2952,
        "difficulty": 2.0
      },
      {
        "id": "79c8102ebd11858d7d67f82030ea0136913ab6be1e572525dc7a4b050643260b",
        "domain": "omni",
        "question": "A regular hexagon has one side along the diameter of a semicircle, and the two opposite vertices on the semicircle. Find the area of the hexagon if the diameter of the semicircle is 1.",
        "answer": "3 \\sqrt{3} / 26",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 678,
        "difficulty": 4.0
      },
      {
        "id": "b6b24a56310f6e75c715a60bc92a70d53fbfb15f0ab90a4c49a26c5a0f837141",
        "domain": "omni",
        "question": "Tetrahedron $A B C D$ with volume 1 is inscribed in circumsphere $\\omega$ such that $A B=A C=A D=2$ and $B C \\cdot C D \\cdot D B=16$. Find the radius of $\\omega$.",
        "answer": "\\frac{5}{3}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2582,
        "difficulty": 5.25
      },
      {
        "id": "05ebd42078cde207e6eb326e7e5df3c8a7b5485b93f178ba8f09fd7288dc1dd3",
        "domain": "omni",
        "question": "Find the number of positive integers $j \\leq 3^{2013}$ such that $$j=\\sum_{k=0}^{m}\\left((-1)^{k} \\cdot 3^{a_{k}}\\right)$$ for some strictly increasing sequence of nonnegative integers $\\left\\{a_{k}\\right\\}$. For example, we may write $3=3^{1}$ and $55=3^{0}-3^{3}+3^{4}$, but 4 cannot be written in this form.",
        "answer": "2^{2013}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 611,
        "difficulty": 5.25
      },
      {
        "id": "06b6f3e301fdd41d1f6e997eccc0e0060035fd281141c45ce4db9c91820071e7",
        "domain": "omni",
        "question": "Nasim buys trading cards in packages of 5 cards and in packages of 8 cards. He can purchase exactly 18 cards by buying two 5-packs and one 8-pack, but he cannot purchase exactly 12 cards with any combination of packages. For how many of the integers $n=24,25,26,27,28,29$ can he buy exactly $n$ cards?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3041,
        "difficulty": 1.5
      },
      {
        "id": "dd1da26eb57a468cad31ded9547783f1206fdcec9042bc86d514cc0845eb8015",
        "domain": "omni",
        "question": "The Red Sox play the Yankees in a best-of-seven series that ends as soon as one team wins four games. Suppose that the probability that the Red Sox win Game $n$ is $\\frac{n-1}{6}$. What is the probability that the Red Sox will win the series?",
        "answer": "1/2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1022,
        "difficulty": 5.0
      },
      {
        "id": "8bf59d72479665c9d44666cba3c405eb30f728e8bc6bc5c416229331ba6cceaa",
        "domain": "omni",
        "question": "In a 3 by 3 grid of unit squares, an up-right path is a path from the bottom left corner to the top right corner that travels only up and right in steps of 1 unit. For such a path $p$, let $A_{p}$ denote the number of unit squares under the path $p$. Compute the sum of $A_{p}$ over all up-right paths $p$.",
        "answer": "90",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 895,
        "difficulty": 4.5
      },
      {
        "id": "37056c6dee449f02f30b0dd247e63ca3fc11153d3b9c278c96e6e7677a7f73b1",
        "domain": "omni",
        "question": "Each square in the following hexomino has side length 1. Find the minimum area of any rectangle that contains the entire hexomino.",
        "answer": "\\frac{21}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 2602,
        "difficulty": 5.0
      },
      {
        "id": "8b494b685398526a2ecc240d75f6e79cfdc23160524e512897d50006e1e881ff",
        "domain": "omni",
        "question": "If you roll four fair 6-sided dice, what is the probability that at least three of them will show the same value?",
        "answer": "\\frac{7}{72}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2272,
        "difficulty": 4.0
      },
      {
        "id": "e83943685871c02359d0f1018816d8a200c665e40f2d92efaf96211be12ef236",
        "domain": "omni",
        "question": "Let \\(A B C\\) be a triangle with \\(\\angle A=18^{\\circ}, \\angle B=36^{\\circ}\\). Let \\(M\\) be the midpoint of \\(A B, D\\) a point on ray \\(C M\\) such that \\(A B=A D ; E\\) a point on ray \\(B C\\) such that \\(A B=B E\\), and \\(F\\) a point on ray \\(A C\\) such that \\(A B=A F\\). Find \\(\\angle F D E\\).",
        "answer": "27",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1673,
        "difficulty": 5.25
      },
      {
        "id": "20f0361cd9062eb66a0f84521712d45b052e20ffe804b6296ba27ea41fe19d3f",
        "domain": "omni",
        "question": "Let $\\pi$ be a randomly chosen permutation of the numbers from 1 through 2012. Find the probability that $\\pi(\\pi(2012))=2012$.",
        "answer": "\\frac{1}{1006}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1963,
        "difficulty": 4.5
      },
      {
        "id": "513566b5ea9815d4d49d62fe77017c52f13e9841ee38cd6f284f84fbfb2055e7",
        "domain": "omni",
        "question": "Suppose $A B C D$ is a convex quadrilateral with $\\angle A B D=105^{\\circ}, \\angle A D B=15^{\\circ}, A C=7$, and $B C=C D=5$. Compute the sum of all possible values of $B D$.",
        "answer": "\\sqrt{291}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1601,
        "difficulty": 5.25
      },
      {
        "id": "ec813baa7dc0a7270e6a06563f0ac9932dfddaff3eeea3010661617c00b94391",
        "domain": "omni",
        "question": "Let $n$ be square with 4 digits, such that all its digits are less than 6. If we add 1 to each digit the resulting number is another square. Find $n$",
        "answer": "2025",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Congruences",
        "source": "cono_sur_olympiad",
        "source_index": 3629,
        "difficulty": 5.0
      },
      {
        "id": "54ceb87ab3eb384eba6b9652c09c19b48fd6f5aeb712b159400bd4f9333439d5",
        "domain": "omni",
        "question": "Pascal has a triangle. In the $n$th row, there are $n+1$ numbers $a_{n, 0}, a_{n, 1}, a_{n, 2}, \\ldots, a_{n, n}$ where $a_{n, 0}=a_{n, n}=1$. For all $1 \\leq k \\leq n-1, a_{n, k}=a_{n-1, k}-a_{n-1, k-1}$. Let $N$ be the value of the sum $$\\sum_{k=0}^{2018} \\frac{\\left|a_{2018, k}\\right|}{\\binom{2018}{k}}$$ Estimate $N$.",
        "answer": "780.9280674537",
        "topic": "Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2172,
        "difficulty": 5.0
      },
      {
        "id": "1ffedc1789b675c082befa9c39086ef080a2f19474a51ca6fb38c529b546e456",
        "domain": "omni",
        "question": "John lists the integers from 1 to 20 in increasing order. He then erases the first half of the integers in the list and rewrites them in order at the end of the second half of the list. Which integer in the new list has exactly 12 integers to its left?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3467,
        "difficulty": 1.0
      },
      {
        "id": "419d573a6e90848629df44b1448daad6cb0ef5c62ab64e677a4c8ef5f058569e",
        "domain": "omni",
        "question": "Evaluate\n\\[\\left \\lfloor  \\ \\prod_{n=1}^{1992} \\frac{3n+2}{3n+1} \\  \\right \\rfloor\\]",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "imo_longlists",
        "source_index": 3936,
        "difficulty": 7.5
      },
      {
        "id": "60e62111d26201d245ca9e55cdb92352a567625279cd4151a3c3d4ffded62264",
        "domain": "omni",
        "question": "An isosceles right triangle $A B C$ has area 1. Points $D, E, F$ are chosen on $B C, C A, A B$ respectively such that $D E F$ is also an isosceles right triangle. Find the smallest possible area of $D E F$.",
        "answer": "\\frac{1}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1964,
        "difficulty": 5.25
      },
      {
        "id": "ce036093770d3f90b4f63dbd9c95c31d97aab02241a42c3ec97aa0fa6bfd77f9",
        "domain": "omni",
        "question": "Unit squares $A B C D$ and $E F G H$ have centers $O_{1}$ and $O_{2}$ respectively, and are originally situated such that $B$ and $E$ are at the same position and $C$ and $H$ are at the same position. The squares then rotate clockwise about their centers at the rate of one revolution per hour. After 5 minutes, what is the area of the intersection of the two squares?",
        "answer": "\\frac{2-\\sqrt{3}}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1820,
        "difficulty": 5.0
      },
      {
        "id": "0301402004f36a6be16056f37bdf8ab4bdf431d742ca0a12ce93e9e95741d516",
        "domain": "omni",
        "question": "Sarah stands at $(0,0)$ and Rachel stands at $(6,8)$ in the Euclidean plane. Sarah can only move 1 unit in the positive $x$ or $y$ direction, and Rachel can only move 1 unit in the negative $x$ or $y$ direction. Each second, Sarah and Rachel see each other, independently pick a direction to move at the same time, and move to their new position. Sarah catches Rachel if Sarah and Rachel are ever at the same point. Rachel wins if she is able to get to $(0,0)$ without being caught; otherwise, Sarah wins. Given that both of them play optimally to maximize their probability of winning, what is the probability that Rachel wins?",
        "answer": "\\[\n\\frac{63}{64}\n\\]",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 344,
        "difficulty": 6.0
      },
      {
        "id": "9f8f32101e6d96e00594a0b86819fccb145f5a21960ddfb9c8efe1a197bd37c9",
        "domain": "omni",
        "question": "A tournament among 2021 ranked teams is played over 2020 rounds. In each round, two teams are selected uniformly at random among all remaining teams to play against each other. The better ranked team always wins, and the worse ranked team is eliminated. Let $p$ be the probability that the second best ranked team is eliminated in the last round. Compute $\\lfloor 2021 p \\rfloor$.",
        "answer": "674",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1002,
        "difficulty": 5.25
      },
      {
        "id": "c40fad79cd4e704c2bd803c3b0941627fd8527ad06fc6c2f4e7649df371b1bd5",
        "domain": "omni",
        "question": "A sequence of real numbers $a_{0}, a_{1}, \\ldots$ is said to be good if the following three conditions hold. (i) The value of $a_{0}$ is a positive integer. (ii) For each non-negative integer $i$ we have $a_{i+1}=2 a_{i}+1$ or $a_{i+1}=\\frac{a_{i}}{a_{i}+2}$. (iii) There exists a positive integer $k$ such that $a_{k}=2014$. Find the smallest positive integer $n$ such that there exists a good sequence $a_{0}, a_{1}, \\ldots$ of real numbers with the property that $a_{n}=2014$.",
        "answer": "60",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series; Mathematics -> Number Theory -> Congruences",
        "source": "apmoapmo_sol",
        "source_index": 1464,
        "difficulty": 7.0
      },
      {
        "id": "907d775a6e254cb1fa488572d22fafd5735dd3993c1e98f74069ce5d5117ddb9",
        "domain": "omni",
        "question": "If a positive integer multiple of 864 is picked randomly, with each multiple having the same probability of being picked, what is the probability that it is divisible by 1944?",
        "answer": "\\frac{1}{9}",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1163,
        "difficulty": 4.0
      },
      {
        "id": "136c8096dc43a58c0abde0b0afaf4184de56506786366a7fb239783e62d93b2d",
        "domain": "omni",
        "question": "Let $A B C D$ be a convex quadrilateral so that all of its sides and diagonals have integer lengths. Given that $\\angle A B C=\\angle A D C=90^{\\circ}, A B=B D$, and $C D=41$, find the length of $B C$.",
        "answer": "580",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1917,
        "difficulty": 3.5
      },
      {
        "id": "8806d96f41b65e24f43f679cb75d3de6ef1cd8eec5a22680326ddf169eeec96b",
        "domain": "omni",
        "question": "Two circles with equal radii intersect as shown. The area of the shaded region equals the sum of the areas of the two unshaded regions. If the area of the shaded region is $216\\pi$, what is the circumference of each circle?",
        "answer": "36\\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "pascal",
        "source_index": 2961,
        "difficulty": 2.0
      },
      {
        "id": "2ca7b50ee4818bd615e644c707e03a38f9de43be79e37d0ace08fc0e843a6a04",
        "domain": "omni",
        "question": "The operation \\( \\otimes \\) is defined by \\( a \\otimes b = \\frac{a}{b} + \\frac{b}{a} \\). What is the value of \\( 4 \\otimes 8 \\)?",
        "answer": "\\frac{5}{2}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 2878,
        "difficulty": 1.0
      },
      {
        "id": "27e603b2a8880ef4e8c1d60e538c6c5f934800efd61f22806de5959ad364d603",
        "domain": "omni",
        "question": "If $x=11$, $y=-8$, and $2x-3z=5y$, what is the value of $z$?",
        "answer": "\\frac{62}{3}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3035,
        "difficulty": 1.0
      },
      {
        "id": "4f93bec48542075ed29c431ec5921e835c6ab6ac6b230ce947b64796364df0c3",
        "domain": "omni",
        "question": "Let $P_1P_2\\ldots P_{24}$ be a regular $24$-sided polygon inscribed in a circle $\\omega$ with circumference $24$. Determine the number of ways to choose sets of eight distinct vertices from these $24$ such that none of the arcs has length $3$ or $8$.",
        "answer": "258",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_national_olympiad",
        "source_index": 203,
        "difficulty": 5.5
      },
      {
        "id": "f0e04a9cf26c8b5adcf1cdc49c264818140d08a4cd35c694879ccc96faec503c",
        "domain": "omni",
        "question": "Joshua chooses five distinct numbers. In how many different ways can he assign these numbers to the variables $p, q, r, s$, and $t$ so that $p<s, q<s, r<t$, and $s<t$?",
        "answer": "8",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "pascal",
        "source_index": 2958,
        "difficulty": 2.0
      },
      {
        "id": "9daa69e57c437d5acfa467b3b8e8628150acc16186dc9966e678dd88a31330e2",
        "domain": "omni",
        "question": "Abbot writes the letter $A$ on the board. Every minute, he replaces every occurrence of $A$ with $A B$ and every occurrence of $B$ with $B A$, hence creating a string that is twice as long. After 10 minutes, there are $2^{10}=1024$ letters on the board. How many adjacent pairs are the same letter?",
        "answer": "341",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1905,
        "difficulty": 5.0
      },
      {
        "id": "66961c680601761b884af17e3d55c73636475a06d7a72ac0be4334daa1e409fe",
        "domain": "omni",
        "question": "Let $\\omega$ be a fixed circle with radius 1, and let $B C$ be a fixed chord of $\\omega$ such that $B C=1$. The locus of the incenter of $A B C$ as $A$ varies along the circumference of $\\omega$ bounds a region $\\mathcal{R}$ in the plane. Find the area of $\\mathcal{R}$.",
        "answer": "\\pi\\left(\\frac{3-\\sqrt{3}}{3}\\right)-1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1730,
        "difficulty": 5.25
      },
      {
        "id": "025ee7499a969a0dfe6c1307be7ad8ea2a90c3a6be3dec34bf607f1337853ad5",
        "domain": "omni",
        "question": "A robotic grasshopper jumps 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After every fourth jump, the grasshopper restarts the sequence of jumps: 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After a total of $n$ jumps, the position of the grasshopper is 162 cm to the west and 158 cm to the south of its original position. What is the sum of the squares of the digits of $n$?",
        "answer": "22",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "cayley",
        "source_index": 2777,
        "difficulty": 1.5
      },
      {
        "id": "255a9537201ba3d818ac779832d44cca4b1d60f01d8c74452e3b4d5b9f34b43d",
        "domain": "omni",
        "question": "Among citizens of Cambridge there exist 8 different types of blood antigens. In a crowded lecture hall are 256 students, each of whom has a blood type corresponding to a distinct subset of the antigens; the remaining of the antigens are foreign to them. Quito the Mosquito flies around the lecture hall, picks a subset of the students uniformly at random, and bites the chosen students in a random order. After biting a student, Quito stores a bit of any antigens that student had. A student bitten while Quito had $k$ blood antigen foreign to him/her will suffer for $k$ hours. What is the expected total suffering of all 256 students, in hours?",
        "answer": "\\frac{2^{135}-2^{128}+1}{2^{119} \\cdot 129}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1053,
        "difficulty": 5.0
      },
      {
        "id": "d71810f4f90b575c68b7ff1136d0cb9f36d3972ca5d71397480ba74374189ed6",
        "domain": "omni",
        "question": "The integers -5 and 6 are shown on a number line. What is the distance between them?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3108,
        "difficulty": 1.0
      },
      {
        "id": "214778f12c08cacd1f01e56d361114eae500b1b41b912b9ee9654153e41cfc94",
        "domain": "omni",
        "question": "Let $f(x)=x^{3}+x+1$. Suppose $g$ is a cubic polynomial such that $g(0)=-1$, and the roots of $g$ are the squares of the roots of $f$. Find $g(9)$.",
        "answer": "899",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1240,
        "difficulty": 5.5
      },
      {
        "id": "1fd5fdf724504aa8c0f10fb143b67c19e6727685d0e3081bcd9947b59a8bb9bd",
        "domain": "omni",
        "question": "Let $n$ be the answer to this problem. An urn contains white and black balls. There are $n$ white balls and at least two balls of each color in the urn. Two balls are randomly drawn from the urn without replacement. Find the probability, in percent, that the first ball drawn is white and the second is black.",
        "answer": "19",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2331,
        "difficulty": 4.0
      },
      {
        "id": "08cc93696708437639ba2b9412e657326df881352adeffb073d43b617f3cf7c4",
        "domain": "omni",
        "question": "Suppose that $d$ is an odd integer and $e$ is an even integer. How many of the following expressions are equal to an odd integer? $d+d, (e+e) \times d, d \times d, d \times(e+d)$",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2896,
        "difficulty": 2.0
      },
      {
        "id": "92f1f5ab3f57db904b580ea8e225e11cc85f63e78a18a8830d36d24cb1a8c217",
        "domain": "omni",
        "question": "How many of the positive divisors of 128 are perfect squares larger than 1?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2759,
        "difficulty": 2.5
      },
      {
        "id": "d8108179e8012f22f6d9360c00d8fd28e3b046b324bdc5f425261eb032bd7c09",
        "domain": "omni",
        "question": "Consider a $5 \\times 5$ grid of squares. Vladimir colors some of these squares red, such that the centers of any four red squares do not form an axis-parallel rectangle (i.e. a rectangle whose sides are parallel to those of the squares). What is the maximum number of squares he could have colored red?",
        "answer": "12",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2089,
        "difficulty": 4.0
      },
      {
        "id": "81b5531bb8ad5fc8baf60e37f6827355031f7eeb6a93f3dcaaa80aaccbfe2744",
        "domain": "omni",
        "question": "Owen spends $\\$ 1.20$ per litre on gasoline. He uses an average of 1 L of gasoline to drive 12.5 km. How much will Owen spend on gasoline to drive 50 km?",
        "answer": "\\$ 4.80",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3514,
        "difficulty": 1.5
      },
      {
        "id": "9bc04c194a9d2b8529bf86e7800deb7e97d1bef96b6a936a6d8a9476c0d87c50",
        "domain": "omni",
        "question": "How many orderings $(a_{1}, \\ldots, a_{8})$ of $(1,2, \\ldots, 8)$ exist such that $a_{1}-a_{2}+a_{3}-a_{4}+a_{5}-a_{6}+a_{7}-a_{8}=0$ ?",
        "answer": "4608",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1383,
        "difficulty": 5.0
      },
      {
        "id": "1c4aa4f8d6ccdf7a69f8d608830b437fac57c29a12de074da39cd5233896098c",
        "domain": "omni",
        "question": "During the weekends, Eli delivers milk in the complex plane. On Saturday, he begins at $z$ and delivers milk to houses located at $z^{3}, z^{5}, z^{7}, \\ldots, z^{2013}$, in that order; on Sunday, he begins at 1 and delivers milk to houses located at $z^{2}, z^{4}, z^{6}, \\ldots, z^{2012}$, in that order. Eli always walks directly (in a straight line) between two houses. If the distance he must travel from his starting point to the last house is $\\sqrt{2012}$ on both days, find the real part of $z^{2}$.",
        "answer": "\\frac{1005}{1006}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1432,
        "difficulty": 4.0
      },
      {
        "id": "b91f97ed576ad9917945c273258163e44fb5c6446bc3264682e4f6c5911011fc",
        "domain": "omni",
        "question": "Find the number of digits in the decimal representation of $2^{41}$.",
        "answer": "13",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "HMMT_11",
        "source_index": 1840,
        "difficulty": 4.0
      },
      {
        "id": "11c422486371fc792664822177e85163f8849cca4a3cc8191613f25fd0dda166",
        "domain": "omni",
        "question": "Let $a, b, c, x, y$, and $z$ be complex numbers such that $a=\\frac{b+c}{x-2}, \\quad b=\\frac{c+a}{y-2}, \\quad c=\\frac{a+b}{z-2}$. If $x y+y z+z x=67$ and $x+y+z=2010$, find the value of $x y z$.",
        "answer": "-5892",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1388,
        "difficulty": 5.5
      },
      {
        "id": "9c535f2fc388c6499e5870b4445c4aa1a51d455288dbdf7bdbbb2876ab7437e0",
        "domain": "omni",
        "question": "Compute $$100^{2}+99^{2}-98^{2}-97^{2}+96^{2}+95^{2}-94^{2}-93^{2}+\\ldots+4^{2}+3^{2}-2^{2}-1^{2}$$",
        "answer": "10100",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2611,
        "difficulty": 3.5
      },
      {
        "id": "844238358efadbb47d1b1e4a7e8fd038a41ae31796542edc87b34af1d46480ce",
        "domain": "omni",
        "question": "In $\\triangle A B C, \\omega$ is the circumcircle, $I$ is the incenter and $I_{A}$ is the $A$-excenter. Let $M$ be the midpoint of arc $\\widehat{B A C}$ on $\\omega$, and suppose that $X, Y$ are the projections of $I$ onto $M I_{A}$ and $I_{A}$ onto $M I$, respectively. If $\\triangle X Y I_{A}$ is an equilateral triangle with side length 1, compute the area of $\\triangle A B C$.",
        "answer": "\\frac{\\sqrt{6}}{7}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 694,
        "difficulty": 5.5
      },
      {
        "id": "a73091ffdcfd14e305c786fd5ab71201652644b498934ab7090a5abe40221d35",
        "domain": "omni",
        "question": "Evaluate \\[ \\sum_{k=1}^\\infty \\frac{(-1)^{k-1}}{k} \\sum_{n=0}^\\infty \\frac{1}{k2^n + 1}. \\]",
        "answer": "1",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Multi-variable; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "putnam",
        "source_index": 3228,
        "difficulty": 8.0
      },
      {
        "id": "84951a8b4217d27a2a0b8cd8482612def3417ffe67b4db04a14dd9b225a5b18a",
        "domain": "omni",
        "question": "How many subsets $S$ of the set $\\{1,2, \\ldots, 10\\}$ satisfy the property that, for all $i \\in[1,9]$, either $i$ or $i+1$ (or both) is in $S$?",
        "answer": "144",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1837,
        "difficulty": 5.0
      },
      {
        "id": "6759cd9e482730b65a6973b3c1a8aa3e303d78a58efc834a86640f533bf092c6",
        "domain": "omni",
        "question": "Let $x<y$ be positive real numbers such that $\\sqrt{x}+\\sqrt{y}=4$ and $\\sqrt{x+2}+\\sqrt{y+2}=5$. Compute $x$.",
        "answer": "\\frac{49}{36}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 836,
        "difficulty": 4.5
      },
      {
        "id": "0f45b1a4eafd7c59f6f5ec783dc5f9a3a10835a0388f493552cc4fe9d2ec8330",
        "domain": "omni",
        "question": "Let $x, y, z$ be real numbers satisfying $$\\frac{1}{x}+y+z=x+\\frac{1}{y}+z=x+y+\\frac{1}{z}=3$$ The sum of all possible values of $x+y+z$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "6106",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2587,
        "difficulty": 5.25
      },
      {
        "id": "8e6311cbc60cbfcb8fff9a3ab37f6041ae85cfce56255d80b21786ea067fcaad",
        "domain": "omni",
        "question": "Let $A, B, C$ be points in that order along a line, such that $A B=20$ and $B C=18$. Let $\\omega$ be a circle of nonzero radius centered at $B$, and let $\\ell_{1}$ and $\\ell_{2}$ be tangents to $\\omega$ through $A$ and $C$, respectively. Let $K$ be the intersection of $\\ell_{1}$ and $\\ell_{2}$. Let $X$ lie on segment $\\overline{K A}$ and $Y$ lie on segment $\\overline{K C}$ such that $X Y \\| B C$ and $X Y$ is tangent to $\\omega$. What is the largest possible integer length for $X Y$?",
        "answer": "35",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1877,
        "difficulty": 5.0
      },
      {
        "id": "cf545afe8bd707e98f136fe2f8254db956a9ffc940f229b841711bafada87a85",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, \\ldots, a_{n}$ be a sequence of distinct positive integers such that $a_{1}+a_{2}+\\cdots+a_{n}=2021$ and $a_{1} a_{2} \\cdots a_{n}$ is maximized. If $M=a_{1} a_{2} \\cdots a_{n}$, compute the largest positive integer $k$ such that $2^{k} \\mid M$.",
        "answer": "62",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 881,
        "difficulty": 5.0
      },
      {
        "id": "266f6345b2ae5703e7c8d7f892eb1498539b63fc52ba1ec463a4e36fd392b4be",
        "domain": "omni",
        "question": "For dinner, Priya is eating grilled pineapple spears. Each spear is in the shape of the quadrilateral PINE, with $P I=6 \\mathrm{~cm}, I N=15 \\mathrm{~cm}, N E=6 \\mathrm{~cm}, E P=25 \\mathrm{~cm}$, and \\angle N E P+\\angle E P I=60^{\\circ}$. What is the area of each spear, in \\mathrm{cm}^{2}$ ?",
        "answer": "\\frac{100 \\sqrt{3}}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2093,
        "difficulty": 4.0
      },
      {
        "id": "b2c6103e00039e7eb0ee1d3789ba7d5708a7c22c6192f5cc30718f0a1fad105c",
        "domain": "omni",
        "question": "Compute the radius of the inscribed circle of a triangle with sides 15,16 , and 17 .",
        "answer": "\\sqrt{21}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 482,
        "difficulty": 3.5
      },
      {
        "id": "da9dcb03d93cbbc69b2c14535883fff88b5ccb33ce48121595371e7d8470ce49",
        "domain": "omni",
        "question": "For $x$ a real number, let $f(x)=0$ if $x<1$ and $f(x)=2 x-2$ if $x \\geq 1$. How many solutions are there to the equation $f(f(f(f(x))))=x ?$",
        "answer": "2",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "HMMT_2",
        "source_index": 421,
        "difficulty": 5.0
      },
      {
        "id": "49ec2ef5bd79e2b4dd41e4e993e1af729873a03bd38047d937bafea083f07f07",
        "domain": "omni",
        "question": "Consider an isosceles triangle $T$ with base 10 and height 12. Define a sequence $\\omega_{1}, \\omega_{2}, \\ldots$ of circles such that $\\omega_{1}$ is the incircle of $T$ and $\\omega_{i+1}$ is tangent to $\\omega_{i}$ and both legs of the isosceles triangle for $i>1$. Find the ratio of the radius of $\\omega_{i+1}$ to the radius of $\\omega_{i}$.",
        "answer": "\\frac{4}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 1696,
        "difficulty": 4.5
      },
      {
        "id": "5bcf9e67b2ff5d909e40f112121e7d75e8a443fdc0af65f27620e3b0a24d1d50",
        "domain": "omni",
        "question": "Let real  $a$, $b$, and $c$ satisfy $$abc+a+b+c=ab+bc+ca+5.$$ Find the least possible value of $a^2+b^2+c^2$.",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "caucasus_mathematical_olympiad",
        "source_index": 3714,
        "difficulty": 6.0
      },
      {
        "id": "9709fd27a3098d8c7b95f11442ba64024be3e315544bdbe9d8c55b43646d51c0",
        "domain": "omni",
        "question": "A frog is at the point $(0,0)$. Every second, he can jump one unit either up or right. He can only move to points $(x, y)$ where $x$ and $y$ are not both odd. How many ways can he get to the point $(8,14)$?",
        "answer": "330",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1298,
        "difficulty": 3.5
      },
      {
        "id": "cd88809a8deffd42f9e7d4a6ef4456954398db67a09dae6a14b6c2514bec37c3",
        "domain": "omni",
        "question": "How many positive integers $n \\leq 20000$ have the properties that $2n$ has 64 positive divisors including 1 and $2n$, and $5n$ has 60 positive divisors including 1 and $5n$?",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "cayley",
        "source_index": 3080,
        "difficulty": 2.5
      },
      {
        "id": "7360796d90fd3982c3fcccfb7390bf3c2913cf0616907371d752cc776499c04a",
        "domain": "omni",
        "question": "Given positive integers $k, m, n$ such that $1 \\leq  k \\leq  m \\leq  n$. Evaluate\n\n\\[\\sum^{n}_{i=0} \\frac{(-1)^i}{n+k+i} \\cdot \\frac{(m+n+i)!}{i!(n-i)!(m+i)!}.\\]",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "china_team_selection_test",
        "source_index": 150,
        "difficulty": 8.0
      },
      {
        "id": "89dfb870d33ca4134baa87fee848f6d2c61d48b981b851be2cd5286d51d54a76",
        "domain": "omni",
        "question": "Suppose that there are initially eight townspeople and one goon. One of the eight townspeople is named Jester. If Jester is sent to jail during some morning, then the game ends immediately in his sole victory. (However, the Jester does not win if he is sent to jail during some night.) Find the probability that only the Jester wins.",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2149,
        "difficulty": 4.0
      },
      {
        "id": "e66c04161efb7d8dfefdab46fdf18497e01d7490151d677dc6cf456360767c92",
        "domain": "omni",
        "question": "Find the smallest possible value of $x+y$ where $x, y \\geq 1$ and $x$ and $y$ are integers that satisfy $x^{2}-29y^{2}=1$",
        "answer": "11621",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 469,
        "difficulty": 5.0
      },
      {
        "id": "253afdd00f806f590d9dfdfc34f762aff0dc92e094622a8c8af0cfeacc38f18c",
        "domain": "omni",
        "question": "Let $x$ be a complex number such that $x+x^{-1}$ is a root of the polynomial $p(t)=t^{3}+t^{2}-2 t-1$. Find all possible values of $x^{7}+x^{-7}$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "HMMT_11",
        "source_index": 1828,
        "difficulty": 3.5
      },
      {
        "id": "0c60d0a10beb69a14f4081ea9f56ed80c7b8a55fd90789e4dbe3b1951de5eeaa",
        "domain": "omni",
        "question": "Points $A, C$, and $B$ lie on a line in that order such that $A C=4$ and $B C=2$. Circles $\\omega_{1}, \\omega_{2}$, and $\\omega_{3}$ have $\\overline{B C}, \\overline{A C}$, and $\\overline{A B}$ as diameters. Circle $\\Gamma$ is externally tangent to $\\omega_{1}$ and $\\omega_{2}$ at $D$ and $E$ respectively, and is internally tangent to $\\omega_{3}$. Compute the circumradius of triangle $C D E$.",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_2",
        "source_index": 566,
        "difficulty": 4.5
      },
      {
        "id": "b9a78aa1a3bb3219a5eccea3e5f124a4ae358fb43e7f9221826d9d9160af2ced",
        "domain": "omni",
        "question": "A teacher must divide 221 apples evenly among 403 students. What is the minimal number of pieces into which she must cut the apples? (A whole uncut apple counts as one piece.)",
        "answer": "611",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 432,
        "difficulty": 4.0
      },
      {
        "id": "7e93685ad2e4dd5ee77086945177ccec8c3edaa7eedf9db2ee018bdeb8091e55",
        "domain": "omni",
        "question": "Let $\\triangle X Y Z$ be a right triangle with $\\angle X Y Z=90^{\\circ}$. Suppose there exists an infinite sequence of equilateral triangles $X_{0} Y_{0} T_{0}, X_{1} Y_{1} T_{1}, \\ldots$ such that $X_{0}=X, Y_{0}=Y, X_{i}$ lies on the segment $X Z$ for all $i \\geq 0, Y_{i}$ lies on the segment $Y Z$ for all $i \\geq 0, X_{i} Y_{i}$ is perpendicular to $Y Z$ for all $i \\geq 0, T_{i}$ and $Y$ are separated by line $X Z$ for all $i \\geq 0$, and $X_{i}$ lies on segment $Y_{i-1} T_{i-1}$ for $i \\geq 1$. Let $\\mathcal{P}$ denote the union of the equilateral triangles. If the area of $\\mathcal{P}$ is equal to the area of $X Y Z$, find $\\frac{X Y}{Y Z}$.",
        "answer": "1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2552,
        "difficulty": 5.0
      },
      {
        "id": "cb6d1aa63fadd3fab3da99bf49ba2947c0c09bcdfd16f5c0e7233cf7d0857875",
        "domain": "omni",
        "question": "A sphere is centered at a point with integer coordinates and passes through the three points $(2,0,0)$, $(0,4,0),(0,0,6)$, but not the origin $(0,0,0)$. If $r$ is the smallest possible radius of the sphere, compute $r^{2}$.",
        "answer": "51",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2191,
        "difficulty": 5.0
      },
      {
        "id": "84fd70b9c55e641b76b7497d69ff02a692b3b20eb41c052d9756c744b77024b4",
        "domain": "omni",
        "question": "Turbo the snail plays a game on a board with $2024$ rows and $2023$ columns. There are hidden monsters in $2022$ of the cells. Initially, Turbo does not know where any of the monsters are, but he knows that there is exactly one monster in each row except the first row and the last row, and that each column contains at most one monster.\n\nTurbo makes a series of attempts to go from the first row to the last row. On each attempt, he chooses to start on any cell in the first row, then repeatedly moves to an adjacent cell sharing a common side. (He is allowed to return to a previously visited cell.) If he reaches a cell with a monster, his attempt ends and he is transported back to the first row to start a new attempt. The monsters do not move, and Turbo remembers whether or not each cell he has visited contains a monster. If he reaches any cell in the last row, his attempt ends and the game is over.\n\nDetermine the minimum value of $n$ for which Turbo has a strategy that guarantees reaching the last row on the $n$-th attempt or earlier, regardless of the locations of the monsters.\n\n[i]",
        "answer": "3",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "imo",
        "source_index": 3792,
        "difficulty": 9.0
      },
      {
        "id": "0de251225a3807bd00dee15b48289e01e057772f07d0af5d532ebc72e3af21f6",
        "domain": "omni",
        "question": "Starting with an empty string, we create a string by repeatedly appending one of the letters $H, M, T$ with probabilities $\\frac{1}{4}, \\frac{1}{2}, \\frac{1}{4}$, respectively, until the letter $M$ appears twice consecutively. What is the expected value of the length of the resulting string?",
        "answer": "6",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1429,
        "difficulty": 5.5
      },
      {
        "id": "93620e731d6c5f7bdc17302d21a8b37a729f2312c5016a7f7c328f285fa3b1af",
        "domain": "omni",
        "question": "For any integer $n$, define $\\lfloor n\\rfloor$ as the greatest integer less than or equal to $n$. For any positive integer $n$, let $$f(n)=\\lfloor n\\rfloor+\\left\\lfloor\\frac{n}{2}\\right\\rfloor+\\left\\lfloor\\frac{n}{3}\\right\\rfloor+\\cdots+\\left\\lfloor\\frac{n}{n}\\right\\rfloor.$$ For how many values of $n, 1 \\leq n \\leq 100$, is $f(n)$ odd?",
        "answer": "55",
        "topic": "Mathematics -> Number Theory -> Divisor Functions -> Other",
        "source": "HMMT_2",
        "source_index": 913,
        "difficulty": 5.0
      },
      {
        "id": "5ea84e74a8788571d5a98ca949b2f2313249cb09906d2ef07f73534b5cf288fc",
        "domain": "omni",
        "question": "There are 100 houses in a row on a street. A painter comes and paints every house red. Then, another painter comes and paints every third house (starting with house number 3) blue. Another painter comes and paints every fifth house red (even if it is already red), then another painter paints every seventh house blue, and so forth, alternating between red and blue, until 50 painters have been by. After this is finished, how many houses will be red?",
        "answer": "52",
        "topic": "Mathematics -> Algebra -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 565,
        "difficulty": 4.5
      },
      {
        "id": "50bbc997766e7d4abdf61fe668e868606e0311f01c1181b8d53666988f3084ac",
        "domain": "omni",
        "question": "Suppose that $x, y$, and $z$ are non-negative real numbers such that $x+y+z=1$. What is the maximum possible value of $x+y^{2}+z^{3}$ ?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 2326,
        "difficulty": 3.5
      },
      {
        "id": "c25c9800650e23113c23173e33fd7dd16c2272e435657825d4136404690c155f",
        "domain": "omni",
        "question": "Consider a cube $A B C D E F G H$, where $A B C D$ and $E F G H$ are faces, and segments $A E, B F, C G, D H$ are edges of the cube. Let $P$ be the center of face $E F G H$, and let $O$ be the center of the cube. Given that $A G=1$, determine the area of triangle $A O P$.",
        "answer": "$\\frac{\\sqrt{2}}{24}$",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 1965,
        "difficulty": 4.5
      },
      {
        "id": "7c541ef3d071960ac582d973bd5c496e2922bde2b7cdb3882032ff75181ec938",
        "domain": "omni",
        "question": "Five points are chosen uniformly at random on a segment of length 1. What is the expected distance between the closest pair of points?",
        "answer": "\\frac{1}{24}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 684,
        "difficulty": 4.5
      },
      {
        "id": "8fdf3780ea848c8190f18ad2222ebb2da7577f882a4cd9791b4ceead850ccca6",
        "domain": "omni",
        "question": "Shuxin begins with 10 red candies, 7 yellow candies, and 3 blue candies. After eating some of the candies, there are equal numbers of red, yellow, and blue candies remaining. What is the smallest possible number of candies that Shuxin ate?",
        "answer": "11",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2784,
        "difficulty": 2.5
      },
      {
        "id": "cafcd3715cf7abe895d9886b886dcd550feed7dd13a3394b876287d23bbd6294",
        "domain": "omni",
        "question": "Bob's Rice ID number has six digits, each a number from 1 to 9, and any digit can be used any number of times. The ID number satisfies the following property: the first two digits is a number divisible by 2, the first three digits is a number divisible by 3, etc. so that the ID number itself is divisible by 6. One ID number that satisfies this condition is 123252. How many different possibilities are there for Bob's ID number?",
        "answer": "324",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1486,
        "difficulty": 4.0
      },
      {
        "id": "d43719e9116d700b1613592549daaa762af3c0729c4b30c1b9ce4a3b74d2b956",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle where $A B=9, B C=10, C A=17$. Let $\\Omega$ be its circumcircle, and let $A_{1}, B_{1}, C_{1}$ be the diametrically opposite points from $A, B, C$, respectively, on $\\Omega$. Find the area of the convex hexagon with the vertices $A, B, C, A_{1}, B_{1}, C_{1}$.",
        "answer": "\\frac{1155}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 861,
        "difficulty": 5.25
      },
      {
        "id": "e9ecd50900087e33a6d7cc5f38e4526008a5aa4e4b6666135569aa45dc1e7ff7",
        "domain": "omni",
        "question": "Let $N$ be a positive integer whose decimal representation contains 11235 as a contiguous substring, and let $k$ be a positive integer such that $10^{k}>N$. Find the minimum possible value of $$ \\frac{10^{k}-1}{\\operatorname{gcd}\\left(N, 10^{k}-1\\right)} $$",
        "answer": "89",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1621,
        "difficulty": 5.0
      },
      {
        "id": "5ac2c1dea442d2a189c716d638f6a5e43af39d1768c96e3ff83d3dccfc8e9158",
        "domain": "omni",
        "question": "The cafeteria in a certain laboratory is open from noon until 2 in the afternoon every Monday for lunch. Two professors eat 15 minute lunches sometime between noon and 2. What is the probability that they are in the cafeteria simultaneously on any given Monday?",
        "answer": "15",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 408,
        "difficulty": 4.0
      },
      {
        "id": "8cbe5d58947b2863a3e6841688ae51e738aa25e0937836d054299eb0e4ae4b7c",
        "domain": "omni",
        "question": "A fair coin is flipped eight times in a row. Let $p$ be the probability that there is exactly one pair of consecutive flips that are both heads and exactly one pair of consecutive flips that are both tails. If $p=\\frac{a}{b}$, where $a, b$ are relatively prime positive integers, compute $100a+b$.",
        "answer": "1028",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2455,
        "difficulty": 5.25
      },
      {
        "id": "86d38484a9d249eb0fcac0c2bc89bc5e7010c0e35b2cf34cec74237aef31b8b8",
        "domain": "omni",
        "question": "In the plane we consider rectangles whose sides are parallel to the coordinate axes and have positive length. Such a rectangle will be called a [i]box[/i]. Two boxes [i]intersect[/i] if they have a common point in their interior or on their boundary. Find the largest $ n$ for which there exist $ n$ boxes $ B_1$, $ \\ldots$, $ B_n$ such that $ B_i$ and $ B_j$ intersect if and only if $ i\\not\\equiv j\\pm 1\\pmod n$.\n\n[i]",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 4264,
        "difficulty": 7.0
      },
      {
        "id": "753bc0cf760f6731475eea67d3f9beebaba1fe5956238f5bb5ea48b80976c352",
        "domain": "omni",
        "question": "In how many ways can you rearrange the letters of \"HMMTHMMT\" such that the consecutive substring \"HMMT\" does not appear?",
        "answer": "361",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1377,
        "difficulty": 5.0
      },
      {
        "id": "cca309022c3bf218b57fea7ad02075bd5602bbfcc118ec870fd9b4eca9ebe11c",
        "domain": "omni",
        "question": "A palindrome is a string that does not change when its characters are written in reverse order. Let S be a 40-digit string consisting only of 0's and 1's, chosen uniformly at random out of all such strings. Let $E$ be the expected number of nonempty contiguous substrings of $S$ which are palindromes. Compute the value of $\\lfloor E\\rfloor$.",
        "answer": "113",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1872,
        "difficulty": 5.0
      },
      {
        "id": "b9a8160fce1b6d45992c2a2b5b1dcca4d88ddfe75adc9eaa2152d4d1917eccb3",
        "domain": "omni",
        "question": "Consider a sequence $x_{n}$ such that $x_{1}=x_{2}=1, x_{3}=\\frac{2}{3}$. Suppose that $x_{n}=\\frac{x_{n-1}^{2} x_{n-2}}{2 x_{n-2}^{2}-x_{n-1} x_{n-3}}$ for all $n \\geq 4$. Find the least $n$ such that $x_{n} \\leq \\frac{1}{10^{6}}$.",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_11",
        "source_index": 2597,
        "difficulty": 5.25
      },
      {
        "id": "46cc0f85a35352e955fdc3ae8f9eff915ac3896162981e22e58581cbf265803a",
        "domain": "omni",
        "question": "Betty has a $3 \\times 4$ grid of dots. She colors each dot either red or maroon. Compute the number of ways Betty can color the grid such that there is no rectangle whose sides are parallel to the grid lines and whose vertices all have the same color.",
        "answer": "408",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2301,
        "difficulty": 4.0
      },
      {
        "id": "d1afe770135c8c56baa7e02dd9f9e60f5112a215211b94a10116f87eb9df59fb",
        "domain": "omni",
        "question": "$A B C$ is a triangle with $A B=15, B C=14$, and $C A=13$. The altitude from $A$ to $B C$ is extended to meet the circumcircle of $A B C$ at $D$. Find $A D$.",
        "answer": "\\frac{63}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1985,
        "difficulty": 4.5
      },
      {
        "id": "25124d1447f4c117a7c44ce31a00178a7e0e6d8c45bd15573a6552af8cda6a53",
        "domain": "omni",
        "question": "For each positive integer $n$, let $a_{n}$ be the smallest nonnegative integer such that there is only one positive integer at most $n$ that is relatively prime to all of $n, n+1, \\ldots, n+a_{n}$. If $n<100$, compute the largest possible value of $n-a_{n}$.",
        "answer": "16",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2083,
        "difficulty": 4.5
      },
      {
        "id": "c01463e72ea819c5a528f325f110cb5e1820f04648e4ae6bdda87835af966986",
        "domain": "omni",
        "question": "In square $PQRS$ with side length 2, each of $P, Q, R$, and $S$ is the centre of a circle with radius 1. What is the area of the shaded region?",
        "answer": "4-\\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "pascal",
        "source_index": 2990,
        "difficulty": 2.0
      },
      {
        "id": "6abcf73f5c520a572a79fb60f4d976cd998044295b581f7afd74b2fccb495847",
        "domain": "omni",
        "question": "64 people are in a single elimination rock-paper-scissors tournament, which consists of a 6-round knockout bracket. Each person has a different rock-paper-scissors skill level, and in any game, the person with the higher skill level will always win. For how many players $P$ is it possible that $P$ wins the first four rounds that he plays?",
        "answer": "49",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2366,
        "difficulty": 4.5
      },
      {
        "id": "cb674c4cd057a16d552d14f3d24c3443c6ebb6a0017434e75e3196341e953fd8",
        "domain": "omni",
        "question": "How many perfect squares divide $10^{10}$?",
        "answer": "36",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1805,
        "difficulty": 4.0
      },
      {
        "id": "60b20eabb3c83134204601bc521c2eaf62870f85a534675efd97f1c387324c08",
        "domain": "omni",
        "question": "Let $\\left(x_{1}, y_{1}\\right), \\ldots,\\left(x_{k}, y_{k}\\right)$ be the distinct real solutions to the equation $$\\left(x^{2}+y^{2}\\right)^{6}=\\left(x^{2}-y^{2}\\right)^{4}=\\left(2 x^{3}-6 x y^{2}\\right)^{3}$$ Then $\\sum_{i=1}^{k}\\left(x_{i}+y_{i}\\right)$ can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "516",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1446,
        "difficulty": 5.25
      },
      {
        "id": "b7904ce394c330033d8fdc01f2b8d459f15265b6a3fbadf0e75fe423a5790ecd",
        "domain": "omni",
        "question": "How many lines pass through exactly two points in the following hexagonal grid?",
        "answer": "60",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1981,
        "difficulty": 5.0
      },
      {
        "id": "cb60994eded33296a528215b53e734038a3259a1dc633d1eaa1b17520c84116d",
        "domain": "omni",
        "question": "In the game of rock-paper-scissors-lizard-Spock, rock defeats scissors and lizard, paper defeats rock and Spock, scissors defeats paper and lizard, lizard defeats paper and Spock, and Spock defeats rock and scissors. If three people each play a game of rock-paper-scissors-lizard-Spock at the same time by choosing one of the five moves at random, what is the probability that one player beats the other two?",
        "answer": "\\frac{12}{25}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2284,
        "difficulty": 4.0
      },
      {
        "id": "b433f27753c7c0223abe4d5e47a127d412744bb5dcb6a138bba4af13e99b907b",
        "domain": "omni",
        "question": "$O$ is the center of square $A B C D$, and $M$ and $N$ are the midpoints of $\\overline{B C}$ and $\\overline{A D}$, respectively. Points $A^{\\prime}, B^{\\prime}, C^{\\prime}, D^{\\prime}$ are chosen on $\\overline{A O}, \\overline{B O}, \\overline{C O}, \\overline{D O}$, respectively, so that $A^{\\prime} B^{\\prime} M C^{\\prime} D^{\\prime} N$ is an equiangular hexagon. The ratio $\\frac{\\left[A^{\\prime} B^{\\prime} M C^{\\prime} D^{\\prime} N\\right]}{[A B C D]}$ can be written as $\\frac{a+b \\sqrt{c}}{d}$, where $a, b, c, d$ are integers, $d$ is positive, $c$ is square-free, and $\\operatorname{gcd}(a, b, d)=1$. Find $1000 a+100 b+10 c+d$.",
        "answer": "8634",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_11",
        "source_index": 2526,
        "difficulty": 5.25
      },
      {
        "id": "51229c62e32463919a5703710cd4d7c2d00e9d5c03740e2474c86d10ccc8503e",
        "domain": "omni",
        "question": "Point $P$ is inside a square $A B C D$ such that $\\angle A P B=135^{\\circ}, P C=12$, and $P D=15$. Compute the area of this square.",
        "answer": "123+6\\sqrt{119}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 712,
        "difficulty": 5.0
      },
      {
        "id": "bcb7f0a9536bc178ecb3afb937d0bf9da4d76736bcd50c58307eb42263f52209",
        "domain": "omni",
        "question": "Suppose that $A, B, C, D$ are four points in the plane, and let $Q, R, S, T, U, V$ be the respective midpoints of $A B, A C, A D, B C, B D, C D$. If $Q R=2001, S U=2002, T V=$ 2003, find the distance between the midpoints of $Q U$ and $R V$.",
        "answer": "2001",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 512,
        "difficulty": 4.5
      },
      {
        "id": "3278e20df98441e217fe9e875a6a15fdff8b6a28beafcc35e8779c7837ca8c80",
        "domain": "omni",
        "question": "Brian has a 20-sided die with faces numbered from 1 to 20, and George has three 6-sided dice with faces numbered from 1 to 6. Brian and George simultaneously roll all their dice. What is the probability that the number on Brian's die is larger than the sum of the numbers on George's dice?",
        "answer": "\\frac{19}{40}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1284,
        "difficulty": 4.0
      },
      {
        "id": "fdf6dd049ceaedf98ff68e4a2eb0fd0f9fb415e0bc1951d174221d5013dcb8ef",
        "domain": "omni",
        "question": "Let $\\mathcal{A}$ denote the set of all polynomials in three variables $x, y, z$ with integer coefficients. Let $\\mathcal{B}$ denote the subset of $\\mathcal{A}$ formed by all polynomials which can be expressed as\n\\begin{align*}\n(x + y + z)P(x, y, z) + (xy + yz + zx)Q(x, y, z) + xyzR(x, y, z)\n\\end{align*}\nwith $P, Q, R \\in \\mathcal{A}$.  Find the smallest non-negative integer $n$ such that $x^i y^j z^k \\in \\mathcal{B}$ for all non-negative integers $i, j, k$ satisfying $i + j + k \\geq n$.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Ring Theory",
        "source": "imo_shortlist",
        "source_index": 3939,
        "difficulty": 9.0
      },
      {
        "id": "cadf43eb09939dd8910015d8b97d81228e464b46df41b619cee0b7bf7e2b8b85",
        "domain": "omni",
        "question": "Denote $S$ as the subset of $\\{1,2,3,\\dots,1000\\}$ with the property that none of the sums of two different elements in $S$ is in $S$. Find the maximum number of elements in $S$.",
        "answer": "501",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "centroamerican",
        "source_index": 3659,
        "difficulty": 6.0
      },
      {
        "id": "b48391fb80e8a9cb47deb82af183c8a3de7a1735db96c44c8355dc9192db4e23",
        "domain": "omni",
        "question": "The equation $x^{2}+2 x=i$ has two complex solutions. Determine the product of their real parts.",
        "answer": "\\frac{1-\\sqrt{2}}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1717,
        "difficulty": 5.25
      },
      {
        "id": "9e4471b34e8dc723670b33baedeefe7416b2753bf51f5252c6cb3af1feedb73a",
        "domain": "omni",
        "question": "Find the greatest common divisor of the numbers $2002+2,2002^{2}+2,2002^{3}+2, \\ldots$.",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 1232,
        "difficulty": 4.0
      },
      {
        "id": "80df6075634434051e18dc3c62c4035d5303c2a4f1c2baffc869c6a23a8b013a",
        "domain": "omni",
        "question": "Let $Q(x)=x^{2}+2x+3$, and suppose that $P(x)$ is a polynomial such that $P(Q(x))=x^{6}+6x^{5}+18x^{4}+32x^{3}+35x^{2}+22x+8$. Compute $P(2)$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2338,
        "difficulty": 5.0
      },
      {
        "id": "14f3c344a37783056e97e718f898368b27e3acbd5243c25a74d636d6fb76ff52",
        "domain": "omni",
        "question": "Mark writes the expression $\\sqrt{d}$ for each positive divisor $d$ of 8 ! on the board. Seeing that these expressions might not be worth points on HMMT, Rishabh simplifies each expression to the form $a \\sqrt{b}$, where $a$ and $b$ are integers such that $b$ is not divisible by the square of a prime number. Compute the sum of $a+b$ across all expressions that Rishabh writes.",
        "answer": "3480",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2467,
        "difficulty": 4.0
      },
      {
        "id": "3a90bb2407cb6a1d295cbf06c204bf84c6582969627f58c9c62773dcf7e72d67",
        "domain": "omni",
        "question": "How many times does the letter \"e\" occur in all problem statements in this year's HMMT February competition?",
        "answer": "1661",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1068,
        "difficulty": 5.25
      },
      {
        "id": "f637af702a930126d0fb162138755366f2f1af80960e55574a11e5cce162a668",
        "domain": "omni",
        "question": "John ate a total of 120 peanuts over four consecutive nights. Each night he ate 6 more peanuts than the night before. How many peanuts did he eat on the fourth night?",
        "answer": "39",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2723,
        "difficulty": 1.5
      },
      {
        "id": "921049f0600a5426a4aab9b6bc0c9730435fa3bab251368a6cb1d3e6018e177d",
        "domain": "omni",
        "question": "Compute $\\sqrt[4]{5508^{3}+5625^{3}+5742^{3}}$, given that it is an integer.",
        "answer": "855",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 710,
        "difficulty": 5.5
      },
      {
        "id": "77f499bd12059ee819b3ee008c30809b97a619c870153fd53ab351d2d6d6d468",
        "domain": "omni",
        "question": "Michael is playing basketball. He makes $10 \\%$ of his shots, and gets the ball back after $90 \\%$ of his missed shots. If he does not get the ball back he stops playing. What is the probability that Michael eventually makes a shot?",
        "answer": "\\frac{10}{19}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1974,
        "difficulty": 3.5
      },
      {
        "id": "809383eadbe4ebb706f60ff5038786646d6fb45e0044bb9ce387d4be9549987b",
        "domain": "omni",
        "question": "Let $$ A=\\lim _{n \\rightarrow \\infty} \\sum_{i=0}^{2016}(-1)^{i} \\cdot \\frac{\\binom{n}{i}\\binom{n}{i+2}}{\\binom{n}{i+1}^{2}} $$ Find the largest integer less than or equal to $\\frac{1}{A}$.",
        "answer": "1",
        "topic": "Mathematics -> Precalculus -> Limits; Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_2",
        "source_index": 1070,
        "difficulty": 5.0
      },
      {
        "id": "b33706e5838fbc4853a3063dd2acec28b1e96b9f8c6434c1ac9da8fd0632b52e",
        "domain": "omni",
        "question": "A stacking of circles in the plane consists of a base, or some number of unit circles centered on the $x$-axis in a row without overlap or gaps, and circles above the $x$-axis that must be tangent to two circles below them (so that if the ends of the base were secured and gravity were applied from below, then nothing would move). How many stackings of circles in the plane have 4 circles in the base?",
        "answer": "14",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_2",
        "source_index": 394,
        "difficulty": 5.0
      },
      {
        "id": "8663acbfc5b2906eeb691bddb0f9962782790c3568003427d85458ce86ada699",
        "domain": "omni",
        "question": "In circle $\\omega$, two perpendicular chords intersect at a point $P$. The two chords have midpoints $M_{1}$ and $M_{2}$ respectively, such that $P M_{1}=15$ and $P M_{2}=20$. Line $M_{1} M_{2}$ intersects $\\omega$ at points $A$ and $B$, with $M_{1}$ between $A$ and $M_{2}$. Compute the largest possible value of $B M_{2}-A M_{1}$.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2462,
        "difficulty": 4.0
      },
      {
        "id": "5a2640ce9dcb6b57687962adbb19ee325d44e975d5d3922379f9f871c7e8e310",
        "domain": "omni",
        "question": "How many ways are there to cover a $3 \\times 8$ rectangle with 12 identical dominoes?",
        "answer": "153",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1448,
        "difficulty": 5.5
      },
      {
        "id": "9e3beb2e0d7f9f389abfb0a897bf24b2f81e82336d5df91566bc35be6fa7f887",
        "domain": "omni",
        "question": "Find the smallest number $n$ such that there exist polynomials $f_1, f_2, \\ldots , f_n$ with rational coefficients satisfying \\[x^2+7 = f_1\\left(x\\right)^2 + f_2\\left(x\\right)^2 + \\ldots + f_n\\left(x\\right)^2.\\]\n\n[i]",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory; Mathematics -> Number Theory -> Factorization",
        "source": "imo_shortlist",
        "source_index": 3839,
        "difficulty": 7.0
      },
      {
        "id": "37e8e575ff71b35999622e29a4d91389026cba66cdc031a1ee92789b9b6942c6",
        "domain": "omni",
        "question": "Suppose $x, y$, and $z$ are real numbers greater than 1 such that $$\\begin{aligned} x^{\\log _{y} z} & =2, \\\\ y^{\\log _{z} x} & =4, \\text { and } \\\\ z^{\\log _{x} y} & =8 \\end{aligned}$$ Compute $\\log _{x} y$.",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_11",
        "source_index": 1870,
        "difficulty": 5.25
      },
      {
        "id": "851968e4c4d923a12d95c21932aa52c9d26669878df66da5b02e716202a8a834",
        "domain": "omni",
        "question": "One writes 268 numbers around a circle, such that the sum of 20 consectutive numbers is always equal to 75. The number 3, 4 and 9 are written in positions 17, 83 and 144 respectively. Find the number in position 210.",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "pan_african MO",
        "source_index": 3730,
        "difficulty": 5.0
      },
      {
        "id": "8bda829c987af6fadaf4a00f8232b87aae74805a729c526559ab76c2ef54dba4",
        "domain": "omni",
        "question": "For a positive integer $n$, let, $\\tau(n)$ be the number of positive integer divisors of $n$. How many integers $1 \\leq n \\leq 50$ are there such that $\\tau(\\tau(n))$ is odd?",
        "answer": "17",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1864,
        "difficulty": 4.0
      },
      {
        "id": "6024a425888357e1c581ca3c07bed27b57ad1d51261834829a1956fd993933b8",
        "domain": "omni",
        "question": "Given a permutation $\\sigma$ of $\\{1,2, \\ldots, 2013\\}$, let $f(\\sigma)$ to be the number of fixed points of $\\sigma$ - that is, the number of $k \\in\\{1,2, \\ldots, 2013\\}$ such that $\\sigma(k)=k$. If $S$ is the set of all possible permutations $\\sigma$, compute $$\\sum_{\\sigma \\in S} f(\\sigma)^{4}$$ (Here, a permutation $\\sigma$ is a bijective mapping from $\\{1,2, \\ldots, 2013\\}$ to $\\{1,2, \\ldots, 2013\\}$.)",
        "answer": "15(2013!)",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1103,
        "difficulty": 5.0
      },
      {
        "id": "f48cda815ab5b3f459a28094bde4d928f0f28ab4ccdb82943008d32af3cf5658",
        "domain": "omni",
        "question": "In triangle $ABC, AB=32, AC=35$, and $BC=x$. What is the smallest positive integer $x$ such that $1+\\cos^{2}A, \\cos^{2}B$, and $\\cos^{2}C$ form the sides of a non-degenerate triangle?",
        "answer": "48",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_11",
        "source_index": 2574,
        "difficulty": 4.0
      },
      {
        "id": "838e74ee16a24087344ff3caf56629e48fdb19276e360ef0e9d752b586c8ffb8",
        "domain": "omni",
        "question": "Compute the number of positive four-digit multiples of 11 whose sum of digits (in base ten) is divisible by 11.",
        "answer": "72",
        "topic": "Mathematics -> Number Theory -> Divisibility -> Other; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2274,
        "difficulty": 4.5
      },
      {
        "id": "7b5a1f6633947540b7e9f053cdae7f70b29cb952177edaa74366c1c0fe12b0a2",
        "domain": "omni",
        "question": "Compute $$\\sum_{n_{60}=0}^{2} \\sum_{n_{59}=0}^{n_{60}} \\cdots \\sum_{n_{2}=0}^{n_{3}} \\sum_{n_{1}=0}^{n_{2}} \\sum_{n_{0}=0}^{n_{1}} 1$$",
        "answer": "1953",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1130,
        "difficulty": 5.0
      },
      {
        "id": "d5ffa31a25f855382ab76a1bd569f375aed47a05076a4b283b5286497c56d613",
        "domain": "omni",
        "question": "Two distinct squares on a $4 \\times 4$ chessboard are chosen, with each pair of squares equally likely to be chosen. A knight is placed on one of the squares. The expected value of the minimum number of moves it takes for the knight to reach the other squarecan be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "1205",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2522,
        "difficulty": 5.0
      },
      {
        "id": "51f4923b3a1159b529fb55b76f954f640da4ea32f6cc04b914c9231de843a00e",
        "domain": "omni",
        "question": "At a nursery, 2006 babies sit in a circle. Suddenly each baby pokes the baby immediately to either its left or its right, with equal probability. What is the expected number of unpoked babies?",
        "answer": "\\frac{1003}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 547,
        "difficulty": 3.5
      },
      {
        "id": "21941c6ce96defe529dc4cd9f52132fbffd20b2e5c4692a1ab0b84e5e41532ed",
        "domain": "omni",
        "question": "The pairwise products $a b, b c, c d$, and $d a$ of positive integers $a, b, c$, and $d$ are $64,88,120$, and 165 in some order. Find $a+b+c+d$.",
        "answer": "42",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 852,
        "difficulty": 5.0
      },
      {
        "id": "333eec4fadb86d5ed1bfb4f8be75aac6d8c9264aa3db7a879641ebdde1bead93",
        "domain": "omni",
        "question": "Jeff has a 50 point quiz at 11 am . He wakes up at a random time between 10 am and noon, then arrives at class 15 minutes later. If he arrives on time, he will get a perfect score, but if he arrives more than 30 minutes after the quiz starts, he will get a 0 , but otherwise, he loses a point for each minute he's late (he can lose parts of one point if he arrives a nonintegral number of minutes late). What is Jeff's expected score on the quiz?",
        "answer": "\\frac{55}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2464,
        "difficulty": 5.0
      },
      {
        "id": "e918a235a4dcaeffd87855031c2ce5ab4f1c1374f7c595e93d61e1313fb7527b",
        "domain": "omni",
        "question": "How many lattice points are enclosed by the triangle with vertices $(0,99),(5,100)$, and $(2003,500) ?$ Don't count boundary points.",
        "answer": "0",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 506,
        "difficulty": 5.0
      },
      {
        "id": "624a432449e8870b3bac3d54c0c4262261ff337be9af87e2dc4388394e825fff",
        "domain": "omni",
        "question": "Let $n$ be the answer to this problem. Suppose square $ABCD$ has side-length 3. Then, congruent non-overlapping squares $EHGF$ and $IHJK$ of side-length $\\frac{n}{6}$ are drawn such that $A, C$, and $H$ are collinear, $E$ lies on $BC$ and $I$ lies on $CD$. Given that $AJG$ is an equilateral triangle, then the area of $AJG$ is $a+b\\sqrt{c}$, where $a, b, c$ are positive integers and $c$ is not divisible by the square of any prime. Find $a+b+c$.",
        "answer": "48",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2258,
        "difficulty": 4.0
      },
      {
        "id": "0f869fb9def8ef0471b69c2c6520f926ac9b9d69f97193815656a9fd33b124e9",
        "domain": "omni",
        "question": "Convex quadrilateral $ ABCD$ is inscribed in a circle, $ \\angle{A}\\equal{}60^o$, $ BC\\equal{}CD\\equal{}1$, rays $ AB$ and $ DC$ intersect at point $ E$, rays $ BC$ and $ AD$ intersect each other at point $ F$. It is given that the perimeters of triangle $ BCE$ and triangle $ CDF$ are both integers. Find the perimeter of quadrilateral $ ABCD$.",
        "answer": "\\frac{38}{7}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "china_team_selection_test",
        "source_index": 92,
        "difficulty": 7.0
      },
      {
        "id": "528a8654a2a0243769a29ac06d36e218b5ffdbb549f5d6e2518fbefa6fc23557",
        "domain": "omni",
        "question": "Suppose that $x$ and $y$ are real numbers that satisfy the two equations: $x^{2} + 3xy + y^{2} = 909$ and $3x^{2} + xy + 3y^{2} = 1287$. What is a possible value for $x+y$?",
        "answer": "27",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 3042,
        "difficulty": 2.5
      },
      {
        "id": "13a2255a45581d336fc9c9ed2338999e3347862bf6cca4f7b1ffa9fba2c78890",
        "domain": "omni",
        "question": "The $GEB$ sequence $1,3,7,12, \\ldots$ is defined by the following properties: (i) the GEB sequence is increasing (that is, each term is larger than the previous term), (ii) the sequence formed using the differences between each pair of consecutive terms in the GEB sequence (namely, the sequence $2,4,5, \\ldots$) is increasing, and (iii) each positive integer that does not occur in the GEB sequence occurs exactly once in the sequence of differences in (ii). What is the 100th term of the GEB sequence?",
        "answer": "5764",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Sequences -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "pascal",
        "source_index": 2918,
        "difficulty": 2.5
      },
      {
        "id": "da13a6c3b64c5dd97b9e1a39bdb442ec536ebc3794a3b9952185292fbfc2196c",
        "domain": "omni",
        "question": "Let $S$ be the set of integers of the form $2^{x}+2^{y}+2^{z}$, where $x, y, z$ are pairwise distinct non-negative integers. Determine the 100th smallest element of $S$.",
        "answer": "577",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1591,
        "difficulty": 5.0
      },
      {
        "id": "c7740416ca67f21a14165a95a5ffd1891dd840e7f6da620282523fa4fd82c589",
        "domain": "omni",
        "question": "Find the sum $$\\frac{2^{1}}{4^{1}-1}+\\frac{2^{2}}{4^{2}-1}+\\frac{2^{4}}{4^{4}-1}+\\frac{2^{8}}{4^{8}-1}+\\cdots$$",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Fraction Series -> Other",
        "source": "HMMT_2",
        "source_index": 946,
        "difficulty": 5.0
      },
      {
        "id": "9a22996d7f5014168d0f45faf110587f07ad7b6c5020a5627ecf3bc66076ae02",
        "domain": "omni",
        "question": "Let $n \\ge 3$ be an integer. What is the largest possible number of interior angles greater than $180^\\circ$ in an $n$-gon in the plane, given that the $n$-gon does not intersect itself and all its sides have the same length?",
        "answer": "0",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "baltic_way",
        "source_index": 3919,
        "difficulty": 5.5
      },
      {
        "id": "4e66893622844c29cb2da9728a1e7f7a76868fc707f42bbac5903d0d83195b63",
        "domain": "omni",
        "question": "If $f(x)$ is a monic quartic polynomial such that $f(-1)=-1, f(2)=-4, f(-3)=-9$, and $f(4)=-16$, find $f(1)$.",
        "answer": "23",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1374,
        "difficulty": 4.0
      },
      {
        "id": "d5bef83c135101fb8d5a447a9610869793dc0e839b0b2180a29ab81ed52e7439",
        "domain": "omni",
        "question": "Bob the bomb-defuser has stumbled upon an active bomb. He opens it up, and finds the red and green wires conveniently located for him to cut. Being a seasoned member of the bomb-squad, Bob quickly determines that it is the green wire that he should cut, and puts his wirecutters on the green wire. But just before he starts to cut, the bomb starts to count down, ticking every second. Each time the bomb ticks, starting at time $t=15$ seconds, Bob panics and has a certain chance to move his wirecutters to the other wire. However, he is a rational man even when panicking, and has a $\\frac{1}{2 t^{2}}$ chance of switching wires at time $t$, regardless of which wire he is about to cut. When the bomb ticks at $t=1$, Bob cuts whatever wire his wirecutters are on, without switching wires. What is the probability that Bob cuts the green wire?",
        "answer": "\\frac{23}{30}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1516,
        "difficulty": 5.0
      },
      {
        "id": "e6a525add9e055e020f65e67f3c58a1ae05187b33452d50d47df06c22cafa455",
        "domain": "omni",
        "question": "Chords $A B$ and $C D$ of a circle are perpendicular and intersect at a point $P$. If $A P=6, B P=12$, and $C D=22$, find the area of the circle.",
        "answer": "130 \\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_11",
        "source_index": 2117,
        "difficulty": 4.0
      },
      {
        "id": "1de34528d215cf41daece2c49b22b84c63930c1d34a1a13c2e38b4c2c91de692",
        "domain": "omni",
        "question": "Suppose there are initially 1001 townspeople and two goons. What is the probability that, when the game ends, there are exactly 1000 people in jail?",
        "answer": "\\frac{3}{1003}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2020,
        "difficulty": 5.0
      },
      {
        "id": "14f2128dbdaa5fda578c979ee5dd2d24a00188d9767ed061ac1da62282b5ff60",
        "domain": "omni",
        "question": "Find the number of eight-digit positive integers that are multiples of 9 and have all distinct digits.",
        "answer": "181440",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2124,
        "difficulty": 4.0
      },
      {
        "id": "bb192f0b0ed5dd3ad17e1d927c5102d874908e4ff168d7bee2642b4e4b327c8a",
        "domain": "omni",
        "question": "Points $P$ and $Q$ are 3 units apart. A circle centered at $P$ with a radius of $\\sqrt{3}$ units intersects a circle centered at $Q$ with a radius of 3 units at points $A$ and $B$. Find the area of quadrilateral APBQ.",
        "answer": "\\frac{3 \\sqrt{11}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 777,
        "difficulty": 4.0
      },
      {
        "id": "3354503f62416b54059bf3ea22e3fd0034014c8d38a331ef97f3aaf2f83a24f5",
        "domain": "omni",
        "question": "Consider an unusual biased coin, with probability $p$ of landing heads, probability $q \\leq p$ of landing tails, and probability \\frac{1}{6}$ of landing on its side (i.e. on neither face). It is known that if this coin is flipped twice, the likelihood that both flips will have the same result is \\frac{1}{2}$. Find $p$.",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1921,
        "difficulty": 3.5
      },
      {
        "id": "98e7ab6c5c17f6447b682ac30aeaac5164409350c6d09206a2bfa2693f173f06",
        "domain": "omni",
        "question": "There are 2017 jars in a row on a table, initially empty. Each day, a nice man picks ten consecutive jars and deposits one coin in each of the ten jars. Later, Kelvin the Frog comes back to see that $N$ of the jars all contain the same positive integer number of coins (i.e. there is an integer $d>0$ such that $N$ of the jars have exactly $d$ coins). What is the maximum possible value of $N$?",
        "answer": "2014",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1674,
        "difficulty": 5.0
      },
      {
        "id": "ce16dd2c7a84634e6c3e29f8d77fdfb4e8f5f58fa888b916c6ce469b42f8578b",
        "domain": "omni",
        "question": "Let $A, B, C, D$ be points chosen on a circle, in that order. Line $B D$ is reflected over lines $A B$ and $D A$ to obtain lines $\\ell_{1}$ and $\\ell_{2}$ respectively. If lines $\\ell_{1}, \\ell_{2}$, and $A C$ meet at a common point and if $A B=4, B C=3, C D=2$, compute the length $D A$.",
        "answer": "\\sqrt{21}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2014,
        "difficulty": 5.0
      },
      {
        "id": "b447f693265a18129ff96bbc218032ae6148ead4a58692d0fc086e480a013521",
        "domain": "omni",
        "question": "What is the value of $m$ if Tobias downloads $m$ apps, each app costs $\\$ 2.00$ plus $10 \\%$ tax, and he spends $\\$ 52.80$ in total on these $m$ apps?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2981,
        "difficulty": 1.0
      },
      {
        "id": "d34a9e766e2499a3a3cfeb669a985abec72b1274495a5dbc04daf9c693f887d5",
        "domain": "omni",
        "question": "2500 chess kings have to be placed on a $100 \\times 100$ chessboard so that\n\n[b](i)[/b] no king can capture any other one (i.e. no two kings are placed in two squares sharing a common vertex);\n[b](ii)[/b] each row and each column contains exactly 25 kings.\n\nFind the number of such arrangements. (Two arrangements differing by rotation or symmetry are supposed to be different.)\n\n[i]",
        "answer": "2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "imo_shortlist",
        "source_index": 4157,
        "difficulty": 9.0
      },
      {
        "id": "80da2be4b7a52da8d8ad7d114087f930980f0ad1d0c7013dffba77d364f664b3",
        "domain": "omni",
        "question": "The numbers $1,2 \\cdots 11$ are arranged in a line from left to right in a random order. It is observed that the middle number is larger than exactly one number to its left. Find the probability that it is larger than exactly one number to its right.",
        "answer": "\\frac{10}{33}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2499,
        "difficulty": 4.0
      },
      {
        "id": "8b8c733f4d3de1a28c76b958cab59a1a1b2e813dd31c1b8ed55b0b9b37ce9850",
        "domain": "omni",
        "question": "Find the number of ways to distribute 4 pieces of candy to 12 children such that no two consecutive children receive candy.",
        "answer": "105",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2399,
        "difficulty": 5.0
      },
      {
        "id": "ca433126d49c9a10f1c344b4751934f492244513719e1128205e595abd40da50",
        "domain": "omni",
        "question": "If \\( x=2 \\) and \\( v=3x \\), what is the value of \\((2v-5)-(2x-5)\\)?",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2695,
        "difficulty": 1.5
      },
      {
        "id": "e1a3ea0a04742966ae77e810f2cbab84d5628f05912d8208c42ef85f4ffa78a3",
        "domain": "omni",
        "question": "Let $S$ be the set of all points in the plane whose coordinates are positive integers less than or equal to 100 (so $S$ has $100^{2}$ elements), and let $\\mathcal{L}$ be the set of all lines $\\ell$ such that $\\ell$ passes through at least two points in $S$. Find, with proof, the largest integer $N \\geq 2$ for which it is possible to choose $N$ distinct lines in $\\mathcal{L}$ such that every two of the chosen lines are parallel.",
        "answer": "4950",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_2",
        "source_index": 225,
        "difficulty": 5.0
      },
      {
        "id": "11f2475fa1da8cd53327492d88040430eb64bfc596f5958a8fa9134796caddb7",
        "domain": "omni",
        "question": "If the system of equations $$\\begin{aligned} & |x+y|=99 \\\\ & |x-y|=c \\end{aligned}$$ has exactly two real solutions $(x, y)$, find the value of $c$.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1047,
        "difficulty": 5.25
      },
      {
        "id": "0bfd4df579cf6aa0539a52c76e5e89a74071d91cbba8e2aea92aee60ff751461",
        "domain": "omni",
        "question": "How many roots does $\\arctan x=x^{2}-1.6$ have, where the arctan function is defined in the range $-\\frac{p i}{2}<\\arctan x<\\frac{p i}{2}$ ?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "HMMT_2",
        "source_index": 407,
        "difficulty": 4.5
      },
      {
        "id": "3540e2b35bbac2947dece412fccafc0aa7cf2db7bc7fb58fbb25e3bbc68b3600",
        "domain": "omni",
        "question": "A real number \\(x\\) is chosen uniformly at random from the interval \\([0,1000]\\). Find the probability that \\(\\left\\lfloor\\frac{\\left\\lfloor\\frac{x}{2.5}\\right\\rfloor}{2.5}\\right\\rfloor=\\left\\lfloor\\frac{x}{6.25}\\right\\rfloor\\).",
        "answer": "\\frac{9}{10}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2370,
        "difficulty": 5.25
      },
      {
        "id": "19423e1477d867f666fa50ab317b0cf9ba594db396942bd3563f24e5c2226278",
        "domain": "omni",
        "question": "At a university dinner, there are 2017 mathematicians who each order two distinct entrées, with no two mathematicians ordering the same pair of entrées. The cost of each entrée is equal to the number of mathematicians who ordered it, and the university pays for each mathematician's less expensive entrée (ties broken arbitrarily). Over all possible sets of orders, what is the maximum total amount the university could have paid?",
        "answer": "127009",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "usa_team_selection_test",
        "source_index": 18,
        "difficulty": 7.0
      },
      {
        "id": "ab21d6c0bc301150952e9feb1fa132eaed99bccd9787bd02423201944bc0049c",
        "domain": "omni",
        "question": "If $\\frac{1}{9}$ of 60 is 5, what is $\\frac{1}{20}$ of 80?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_2",
        "source_index": 383,
        "difficulty": 3.5
      },
      {
        "id": "5d49a4ec4f04eb4d17e54b4a69c6e588b58d9eb4f5f999988f0ae680fbae3e9d",
        "domain": "omni",
        "question": "A rubber band is 4 inches long. An ant begins at the left end. Every minute, the ant walks one inch along rightwards along the rubber band, but then the band is stretched (uniformly) by one inch. For what value of $n$ will the ant reach the right end during the $n$th minute?",
        "answer": "7",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 3321,
        "difficulty": 3.5
      },
      {
        "id": "4da640e9ab4715bf9862f345f93a1aae220077293357f017da0855ba1d1ba638",
        "domain": "omni",
        "question": "In the $x-y$ plane, draw a circle of radius 2 centered at $(0,0)$. Color the circle red above the line $y=1$, color the circle blue below the line $y=-1$, and color the rest of the circle white. Now consider an arbitrary straight line at distance 1 from the circle. We color each point $P$ of the line with the color of the closest point to $P$ on the circle. If we pick such an arbitrary line, randomly oriented, what is the probability that it contains red, white, and blue points?",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other; Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_2",
        "source_index": 818,
        "difficulty": 5.0
      },
      {
        "id": "745f18242ffc75e957680fbbbbf89829e41e22b81c4d15d0c1bea6ad8327ef49",
        "domain": "omni",
        "question": "Every second, Andrea writes down a random digit uniformly chosen from the set $\\{1,2,3,4\\}$. She stops when the last two numbers she has written sum to a prime number. What is the probability that the last number she writes down is 1?",
        "answer": "15/44",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1065,
        "difficulty": 4.0
      },
      {
        "id": "8831dd47bb6e97178378b71b9c67f1c054bbaa206a99b7e7dd439f117e14b62f",
        "domain": "omni",
        "question": "Amina and Bert alternate turns tossing a fair coin. Amina goes first and each player takes three turns. The first player to toss a tail wins. If neither Amina nor Bert tosses a tail, then neither wins. What is the probability that Amina wins?",
        "answer": "\\frac{21}{32}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "fermat",
        "source_index": 2740,
        "difficulty": 2.5
      },
      {
        "id": "e1d03781fdb57aae0f9b3e9627e25dba137fb2ce697efb3862beb1ad155c53f7",
        "domain": "omni",
        "question": "An ordered pair of sets $(A, B)$ is good if $A$ is not a subset of $B$ and $B$ is not a subset of $A$. How many ordered pairs of subsets of $\\{1,2, \\ldots, 2017\\}$ are good?",
        "answer": "4^{2017}-2 \\cdot 3^{2017}+2^{2017}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 431,
        "difficulty": 5.0
      },
      {
        "id": "640311f3d02a59a5730c816a7c2437f4325848fbdce2e32c83fffa68a4bf38de",
        "domain": "omni",
        "question": "If $2 \\times 2 \\times 3 \\times 3 \\times 5 \\times 6=5 \\times 6 \\times n \\times n$, what is a possible value of $n$?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2905,
        "difficulty": 1.5
      },
      {
        "id": "443f3f708b765d04dc0fd8673622ed3ce55fbac94814a2e0447f0bc2d69c0bed",
        "domain": "omni",
        "question": "Paul fills in a $7 \\times 7$ grid with the numbers 1 through 49 in a random arrangement. He then erases his work and does the same thing again (to obtain two different random arrangements of the numbers in the grid). What is the expected number of pairs of numbers that occur in either the same row as each other or the same column as each other in both of the two arrangements?",
        "answer": "147 / 2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1319,
        "difficulty": 5.0
      },
      {
        "id": "f5644890ec7b4dbdf14a0c657c45f02892ab8a55ea98b74fe88d2b61323383b9",
        "domain": "omni",
        "question": "Let $n$ be the answer to this problem. Hexagon $ABCDEF$ is inscribed in a circle of radius 90. The area of $ABCDEF$ is $8n$, $AB=BC=DE=EF$, and $CD=FA$. Find the area of triangle $ABC$.",
        "answer": "2592",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2182,
        "difficulty": 4.0
      },
      {
        "id": "7f22020d33f4b8f84d529e2bbcbebc3a4c4700012d452622e969afe562a27c60",
        "domain": "omni",
        "question": "The integers $a, b$ and $c$ satisfy the equations $a+5=b$, $5+b=c$, and $b+c=a$. What is the value of $b$?",
        "answer": "-10",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2835,
        "difficulty": 2.0
      },
      {
        "id": "f39fac801eec26798d735fc3409c12806522941a0e4aa20a9a9f66c8042f3253",
        "domain": "omni",
        "question": "The three numbers $5, a, b$ have an average (mean) of 33. What is the average of $a$ and $b$?",
        "answer": "47",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2668,
        "difficulty": 1.5
      },
      {
        "id": "ef257c12c66ea8f268d5da7c0ba3ff683a8b642717564cd0c91a642580e74eed",
        "domain": "omni",
        "question": "Victor has a drawer with two red socks, two green socks, two blue socks, two magenta socks, two lavender socks, two neon socks, two mauve socks, two wisteria socks, and 2000 copper socks, for a total of 2016 socks. He repeatedly draws two socks at a time from the drawer at random, and stops if the socks are of the same color. However, Victor is red-green colorblind, so he also stops if he sees a red and green sock. What is the probability that Victor stops with two socks of the same color? Assume Victor returns both socks to the drawer at each step.",
        "answer": "\\frac{1999008}{1999012}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 994,
        "difficulty": 5.0
      },
      {
        "id": "6c6fbaacbfc7b80d9ffcc435ff7219568d2c4947020bf99c28474ac97f9b68a7",
        "domain": "omni",
        "question": "Let $p=2^{24036583}-1$, the largest prime currently known. For how many positive integers $c$ do the quadratics \\pm x^{2} \\pm p x \\pm c all have rational roots?",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1004,
        "difficulty": 4.0
      },
      {
        "id": "d4ef07465f38574057c934f7ebe3a9bfa83a0ccc23944850d53ad77c8cd0eefd",
        "domain": "omni",
        "question": "How many functions $f:\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ have the property that $f(\\{1,2,3\\})$ and $f(f(\\{1,2,3\\}))$ are disjoint?",
        "answer": "94",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 2386,
        "difficulty": 5.0
      },
      {
        "id": "3fad378ebb9d9253ac1951daa91fbaf41c7fa0dce74ed272ccdd57881104ee97",
        "domain": "omni",
        "question": "What is the earliest row in which the number 2004 may appear?",
        "answer": "12",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1325,
        "difficulty": 4.5
      },
      {
        "id": "738360434c3c4a3cef38593833e6e4e418786725b7df3911f0241568ac8bb573",
        "domain": "omni",
        "question": "Find the number of solutions in positive integers $(k ; a_{1}, a_{2}, \\ldots, a_{k} ; b_{1}, b_{2}, \\ldots, b_{k})$ to the equation $$a_{1}(b_{1})+a_{2}(b_{1}+b_{2})+\\cdots+a_{k}(b_{1}+b_{2}+\\cdots+b_{k})=7$$",
        "answer": "15",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 728,
        "difficulty": 4.5
      },
      {
        "id": "fca8953661e1357020777bd2ae3dfb5f8b5aec7f387a1879f02ec64d15c766ce",
        "domain": "omni",
        "question": "Given a $9 \\times 9$ chess board, we consider all the rectangles whose edges lie along grid lines (the board consists of 81 unit squares, and the grid lines lie on the borders of the unit squares). For each such rectangle, we put a mark in every one of the unit squares inside it. When this process is completed, how many unit squares will contain an even number of marks?",
        "answer": "56",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1127,
        "difficulty": 4.0
      },
      {
        "id": "ee55705ff292a5b55d3562a7cf0c8c614af6504166fa8fa3a87579e27d95e5d2",
        "domain": "omni",
        "question": "How many ways are there to insert +'s between the digits of 111111111111111 (fifteen 1's) so that the result will be a multiple of 30?",
        "answer": "2002",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1461,
        "difficulty": 4.0
      },
      {
        "id": "2dfeda062e2dd6da6d87204a03b3e32f82f465f276076307dc77ffed3c682cb5",
        "domain": "omni",
        "question": "Several positive integers are written on a blackboard. The sum of any two of them is some power of two (for example, $2, 4, 8,...$). What is the maximal possible number of different integers on the blackboard?",
        "answer": "2",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "ToT",
        "source_index": 3756,
        "difficulty": 5.0
      },
      {
        "id": "57edab0133676d9f8e66e581198d8709dea63afcccb5b556fa4ab4a1901e4f75",
        "domain": "omni",
        "question": "For each prime $p$, a polynomial $P(x)$ with rational coefficients is called $p$-good if and only if there exist three integers $a, b$, and $c$ such that $0 \\leq a<b<c<\\frac{p}{3}$ and $p$ divides all the numerators of $P(a)$, $P(b)$, and $P(c)$, when written in simplest form. Compute the number of ordered pairs $(r, s)$ of rational numbers such that the polynomial $x^{3}+10x^{2}+rx+s$ is $p$-good for infinitely many primes $p$.",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Congruences; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 716,
        "difficulty": 7.0
      },
      {
        "id": "d4432064deb8242a7020e2f612abeaa51a7d0e0dc218708403059a469533a1e2",
        "domain": "omni",
        "question": "Consider the cube whose vertices are the eight points $(x, y, z)$ for which each of $x, y$, and $z$ is either 0 or 1 . How many ways are there to color its vertices black or white such that, for any vertex, if all of its neighbors are the same color then it is also that color? Two vertices are neighbors if they are the two endpoints of some edge of the cube.",
        "answer": "118",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 739,
        "difficulty": 5.0
      },
      {
        "id": "0e934a619da066d274c63a4564bb07b26591bb84b8250e5af178a6cc4b07b51d",
        "domain": "omni",
        "question": "The average of 1, 3, and \\( x \\) is 3. What is the value of \\( x \\)?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2766,
        "difficulty": 1.5
      },
      {
        "id": "98beb11839f0a6eae5d2378a49f1e18103648ce02d65a0836f9f9d49a927c227",
        "domain": "omni",
        "question": "Call a string of letters $S$ an almost palindrome if $S$ and the reverse of $S$ differ in exactly two places. Find the number of ways to order the letters in $H M M T T H E M E T E A M$ to get an almost palindrome.",
        "answer": "2160",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1791,
        "difficulty": 5.0
      },
      {
        "id": "203747c47a34f7833ba688038beb414406f225e01a882f10f62f6376ed031f04",
        "domain": "omni",
        "question": "The sequence $\\left\\{a_{n}\\right\\}_{n \\geq 1}$ is defined by $a_{n+2}=7 a_{n+1}-a_{n}$ for positive integers $n$ with initial values $a_{1}=1$ and $a_{2}=8$. Another sequence, $\\left\\{b_{n}\\right\\}$, is defined by the rule $b_{n+2}=3 b_{n+1}-b_{n}$ for positive integers $n$ together with the values $b_{1}=1$ and $b_{2}=2$. Find \\operatorname{gcd}\\left(a_{5000}, b_{501}\\right).",
        "answer": "89",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1561,
        "difficulty": 5.25
      },
      {
        "id": "9d874654257e35b448ca02ee62fa5dc57786ae015792bd7649570d2f2d923db2",
        "domain": "omni",
        "question": "Trapezoid $A B C D$, with bases $A B$ and $C D$, has side lengths $A B=28, B C=13, C D=14$, and $D A=15$. Let diagonals $A C$ and $B D$ intersect at $P$, and let $E$ and $F$ be the midpoints of $A P$ and $B P$, respectively. Find the area of quadrilateral $C D E F$.",
        "answer": "112",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2614,
        "difficulty": 5.0
      },
      {
        "id": "e56856a05f78b87ed859ed04cd6d166ca561c4248b9d8fc1e82b0a703b2d0042",
        "domain": "omni",
        "question": "Simplify: $2 \\sqrt{1.5+\\sqrt{2}}-(1.5+\\sqrt{2})$.",
        "answer": "1/2",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 3350,
        "difficulty": 5.0
      },
      {
        "id": "5b383e586cf83a766aa3d1c81a09e7c90ebe462e19b34ac085a583492bbb34b2",
        "domain": "omni",
        "question": "For any positive real number $x, \\lfloor x \\rfloor$ denotes the largest integer less than or equal to $x$. If $\\lfloor x \\rfloor \\cdot x = 36$ and $\\lfloor y \\rfloor \\cdot y = 71$ where $x, y > 0$, what is $x + y$ equal to?",
        "answer": "\\frac{119}{8}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "fermat",
        "source_index": 2779,
        "difficulty": 2.5
      },
      {
        "id": "095835057c5f2cb6db6e46d443b477fcec20485829df578bb299e7f154583be2",
        "domain": "omni",
        "question": "Suppose $A_1,A_2,\\cdots ,A_n \\subseteq \\left \\{ 1,2,\\cdots ,2018 \\right \\}$ and $\\left | A_i \\right |=2, i=1,2,\\cdots ,n$, satisfying that $$A_i + A_j, \\; 1 \\le i \\le j \\le n ,$$ are distinct from each other. $A + B = \\left \\{ a+b|a\\in A,\\,b\\in B \\right \\}$. Determine the maximal value of $n$.",
        "answer": "4033",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 196,
        "difficulty": 8.0
      },
      {
        "id": "628312fadf26dacdaacecac97e9b4eb6fa13b917e1bd03067a3fc4d9d21ebeac",
        "domain": "omni",
        "question": "If $x, y, z$ are real numbers such that $xy=6, x-z=2$, and $x+y+z=9$, compute $\\frac{x}{y}-\\frac{z}{x}-\\frac{z^{2}}{xy}$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2460,
        "difficulty": 4.5
      },
      {
        "id": "e5be9686d7d94aaddcf0750181560724e71409d4642b6889edc55cb7741f00b0",
        "domain": "omni",
        "question": "Create a cube $C_{1}$ with edge length 1. Take the centers of the faces and connect them to form an octahedron $O_{1}$. Take the centers of the octahedron's faces and connect them to form a new cube $C_{2}$. Continue this process infinitely. Find the sum of all the surface areas of the cubes and octahedrons.",
        "answer": "\\frac{54+9 \\sqrt{3}}{8}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2062,
        "difficulty": 5.0
      },
      {
        "id": "fce2a63653280cd1f49935c8d6174d0c9f5613915c93a481de9d23de4b9aef5c",
        "domain": "omni",
        "question": "Jeffrey writes the numbers 1 and $100000000=10^{8}$ on the blackboard. Every minute, if $x, y$ are on the board, Jeffrey replaces them with $\\frac{x+y}{2} \\text{ and } 2\\left(\\frac{1}{x}+\\frac{1}{y}\\right)^{-1}$. After 2017 minutes the two numbers are $a$ and $b$. Find $\\min (a, b)$ to the nearest integer.",
        "answer": "10000",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 435,
        "difficulty": 5.25
      },
      {
        "id": "6f0103974572b7661651e18432b432eec0e0ee0dc9a3507075c5dd60ff8ed7c2",
        "domain": "omni",
        "question": "There are functions $f(x)$ with the following properties: $f(x)=ax^{2}+bx+c$ for some integers $a, b$ and $c$ with $a>0$, and $f(p)=f(q)=17$ and $f(p+q)=47$ for some prime numbers $p$ and $q$ with $p<q$. For each such function, the value of $f(pq)$ is calculated. The sum of all possible values of $f(pq)$ is $S$. What are the rightmost two digits of $S$?",
        "answer": "71",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions; Mathematics -> Number Theory -> Prime Numbers",
        "source": "fermat",
        "source_index": 2830,
        "difficulty": 2.5
      },
      {
        "id": "1b9a232ad0b83540a7e40e78ad0ce40cc1a16701295739624304d6f382b006a7",
        "domain": "omni",
        "question": "For each positive integer $n$, let $s(n)$ be the sum of the squares of the digits of $n$. For example, $s(15)=1^2+5^2=26$. Determine all integers $n\\geq 1$ such that $s(n)=n$.",
        "answer": "1",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "bero_American",
        "source_index": 3722,
        "difficulty": 5.0
      },
      {
        "id": "2d24ba1e41c8ae9a80a44d547e7f885a0a3a196be266617d132894eb9a126781",
        "domain": "omni",
        "question": "In Rad's garden there are exactly 30 red roses, exactly 19 yellow roses, and no other roses. How many of the yellow roses does Rad need to remove so that $\\frac{2}{7}$ of the roses in the garden are yellow?",
        "answer": "7",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3058,
        "difficulty": 2.0
      },
      {
        "id": "b230e29c11c3e561673f0788f7c2774ec4067336d83dbe34cc71259067a911a1",
        "domain": "omni",
        "question": "Find a real number $t$ such that for any set of 120 points $P_1, \\ldots P_{120}$ on the boundary of a unit square, there exists a point $Q$ on this boundary with $|P_1Q| + |P_2Q| + \\cdots + |P_{120}Q| = t$.",
        "answer": "30(1 + \\sqrt{5})",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Other; Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "usa_team_selection_test",
        "source_index": 16,
        "difficulty": 8.0
      },
      {
        "id": "9ee8db9295b6cf8a9b3037fba400db52763f058c8a0e2991fc415ac5de28a1b6",
        "domain": "omni",
        "question": "Find the area of a triangle with side lengths 14, 48, and 50.",
        "answer": "336",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1741,
        "difficulty": 3.5
      },
      {
        "id": "0c30f6a6893d2950df43ccacc1eb3f2512817ff782f2855096c6360f9950d590",
        "domain": "omni",
        "question": "A circle is tangent to both branches of the hyperbola $x^{2}-20y^{2}=24$ as well as the $x$-axis. Compute the area of this circle.",
        "answer": "504\\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Calculus -> Differential Calculus -> Related Rates",
        "source": "HMMT_2",
        "source_index": 699,
        "difficulty": 5.0
      },
      {
        "id": "1dced44b7b4b7b35bee062333b0cead6f246a07962f59a83fa6da6897a36410b",
        "domain": "omni",
        "question": "An infinite sequence of positive real numbers is defined by \\( a_{0}=1 \\) and \\( a_{n+2}=6 a_{n}-a_{n+1} \\) for \\( n=0,1,2, \\cdots \\) Find the possible value(s) of \\( a_{2007} \\).",
        "answer": "2^{2007}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1371,
        "difficulty": 5.0
      },
      {
        "id": "51f17065a5520d99be6f23a1cd48ed795c71d34b4caf9810bb52ccdf274dfa94",
        "domain": "omni",
        "question": "Find the largest positive integer \\(n\\) for which there exist \\(n\\) finite sets \\(X_{1}, X_{2}, \\ldots, X_{n}\\) with the property that for every \\(1 \\leq a<b<c \\leq n\\), the equation \\(\\left|X_{a} \\cup X_{b} \\cup X_{c}\\right|=\\lceil\\sqrt{a b c}\\rceil\\) holds.",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "HMMT_2",
        "source_index": 1560,
        "difficulty": 5.0
      },
      {
        "id": "9c128bc88ee78215d787341fa528f51120fefbf945855f8d5c5051583b1053f3",
        "domain": "omni",
        "question": "FIx positive integer $n$. Prove: For any positive integers $a,b,c$ not exceeding $3n^2+4n$, there exist integers $x,y,z$ with absolute value not exceeding $2n$ and not all $0$, such that $ax+by+cz=0$",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Linear Transformations",
        "source": "china_team_selection_test",
        "source_index": 303,
        "difficulty": 9.0
      },
      {
        "id": "e81477c7e2d73ec299432d53e596f40c196e882f7dfde351d4edfbbb8b773e62",
        "domain": "omni",
        "question": "Compute the sum of all positive integers $n$ such that $50 \\leq n \\leq 100$ and $2 n+3$ does not divide $2^{n!}-1$.",
        "answer": "222",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1638,
        "difficulty": 5.5
      },
      {
        "id": "0f3682bc1ad44a50201dee3caf55ec86e0957070f069423b2b81d6f7b206f6d1",
        "domain": "omni",
        "question": "There are 21 competitors with distinct skill levels numbered $1,2, \\ldots, 21$. They participate in a pingpong tournament as follows. First, a random competitor is chosen to be \"active\", while the rest are \"inactive.\" Every round, a random inactive competitor is chosen to play against the current active one. The player with the higher skill will win and become (or remain) active, while the loser will be eliminated from the tournament. The tournament lasts for 20 rounds, after which there will only be one player remaining. Alice is the competitor with skill 11. What is the expected number of games that she will get to play?",
        "answer": "\\frac{47}{42}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2256,
        "difficulty": 4.0
      },
      {
        "id": "60be417c0fc0fade0d4e340e18f3e3cf34a78ec5834a1ba03a56688a11396e42",
        "domain": "omni",
        "question": "How many integers between 1 and 2000 inclusive share no common factors with 2001?",
        "answer": "1232",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 1280,
        "difficulty": 4.0
      },
      {
        "id": "ec1ad439b8924305635fc3649eca5233dd10a9f9a38fc14dd9e5d6defcaf7302",
        "domain": "omni",
        "question": "Chris received a mark of $50 \\%$ on a recent test. Chris answered 13 of the first 20 questions correctly. Chris also answered $25 \\%$ of the remaining questions on the test correctly. If each question on the test was worth one mark, how many questions in total were on the test?",
        "answer": "32",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2887,
        "difficulty": 2.0
      },
      {
        "id": "5d9f443dcd49d94c75366c239581b3dda7174a3333b2a2ea09dd42f170b093fb",
        "domain": "omni",
        "question": "Let $ABCD$ be a trapezoid with $AB \\parallel CD$ and $\\angle D=90^{\\circ}$. Suppose that there is a point $E$ on $CD$ such that $AE=BE$ and that triangles $AED$ and $CEB$ are similar, but not congruent. Given that $\\frac{CD}{AB}=2014$, find $\\frac{BC}{AD}$.",
        "answer": "\\sqrt{4027}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 873,
        "difficulty": 4.5
      },
      {
        "id": "1392bc46e769c5e77fbf5b43501131df0c830ddd0c3892a1f497c395acf67f9f",
        "domain": "omni",
        "question": "Determine the smallest positive integer $n \\geq 3$ for which $$A \\equiv 2^{10 n}\\left(\\bmod 2^{170}\\right)$$ where $A$ denotes the result when the numbers $2^{10}, 2^{20}, \\ldots, 2^{10 n}$ are written in decimal notation and concatenated (for example, if $n=2$ we have $A=10241048576$).",
        "answer": "14",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 1320,
        "difficulty": 4.5
      },
      {
        "id": "46b48e37ce00a4459f3cf3ccf5dd1c01c0a95a5324030305948eca607ec705c0",
        "domain": "omni",
        "question": "A real number $x$ satisfies $9^{x}+3^{x}=6$. Compute the value of $16^{1 / x}+4^{1 / x}$.",
        "answer": "90",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 1928,
        "difficulty": 4.0
      },
      {
        "id": "552697b6fe4c2cdabdee9de94fc329c20575f636acea127595d98b71e75e15d2",
        "domain": "omni",
        "question": "The remainder when 111 is divided by 10 is 1. The remainder when 111 is divided by the positive integer $n$ is 6. How many possible values of $n$ are there?",
        "answer": "5",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2773,
        "difficulty": 2.0
      },
      {
        "id": "c5388f98d92ef51b75f3e7f185b340b968d73e3040e842db72ed97bae68ef5a5",
        "domain": "omni",
        "question": "In the figure below, how many ways are there to select 5 bricks, one in each row, such that any two bricks in adjacent rows are adjacent?",
        "answer": "61",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1224,
        "difficulty": 4.0
      },
      {
        "id": "246e7f297d00410c7146ed2c804080bf2602ee06a7e7ea30823b2562a3037978",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, \\ldots$ be an arithmetic sequence and $b_{1}, b_{2}, \\ldots$ be a geometric sequence. Suppose that $a_{1} b_{1}=20$, $a_{2} b_{2}=19$, and $a_{3} b_{3}=14$. Find the greatest possible value of $a_{4} b_{4}$.",
        "answer": "\\frac{37}{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1558,
        "difficulty": 5.5
      },
      {
        "id": "9f082686503bb94cda1e241094e9c45647cf4649b96a9c69b7056c49f1448a58",
        "domain": "omni",
        "question": "Call a triangle nice if the plane can be tiled using congruent copies of this triangle so that any two triangles that share an edge (or part of an edge) are reflections of each other via the shared edge. How many dissimilar nice triangles are there?",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1912,
        "difficulty": 5.0
      },
      {
        "id": "689cba6ec857fa045249e5fbd8c5a7a2f7b38ba2eb1f8ad47a5b17df6761305b",
        "domain": "omni",
        "question": "Pablo has 27 solid $1 \\times 1 \\times 1$ cubes that he assembles in a larger $3 \\times 3 \\times 3$ cube. If 10 of the smaller cubes are red, 9 are blue, and 8 are yellow, what is the smallest possible surface area of the larger cube that is red?",
        "answer": "12",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "pascal",
        "source_index": 3175,
        "difficulty": 2.25
      },
      {
        "id": "f6a28d8f9c74c21ff577b46c3d04aab037fdd8456b6ee835afd02beff9ef8efe",
        "domain": "omni",
        "question": "Determine the number of 8-tuples of nonnegative integers $\\left(a_{1}, a_{2}, a_{3}, a_{4}, b_{1}, b_{2}, b_{3}, b_{4}\\right)$ satisfying $0 \\leq a_{k} \\leq k$, for each $k=1,2,3,4$, and $a_{1}+a_{2}+a_{3}+a_{4}+2 b_{1}+3 b_{2}+4 b_{3}+5 b_{4}=19$.",
        "answer": "1540",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1640,
        "difficulty": 5.0
      },
      {
        "id": "c7c2df8582a1363a626e2ce821999c68c619e02b2384744dbde8b9b63c08dacc",
        "domain": "omni",
        "question": "We have an $n$-gon, and each of its vertices is labeled with a number from the set $\\{1, \\ldots, 10\\}$. We know that for any pair of distinct numbers from this set there is at least one side of the polygon whose endpoints have these two numbers. Find the smallest possible value of $n$.",
        "answer": "50",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 674,
        "difficulty": 5.25
      },
      {
        "id": "93b01b3c9060ec9884ce7e2b0b84cb83e38aa76eaff9b17360715c910d46b8ca",
        "domain": "omni",
        "question": "In the sum shown, $P, Q$ and $R$ represent three different single digits. What is the value of $P+Q+R$?\n\n\\begin{tabular}{r}\n$P 7 R$ \\\\\n$+\\quad 39 R$ \\\\\n\\hline$R Q 0$\n\\end{tabular}",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Number Theory -> Prime Numbers",
        "source": "fermat",
        "source_index": 3040,
        "difficulty": 2.0
      },
      {
        "id": "fc749b39b34486e0ccde936a3c8693cec55ea4864c0e17f488f4b0d05b8f16fb",
        "domain": "omni",
        "question": "Mona has 12 match sticks of length 1, and she has to use them to make regular polygons, with each match being a side or a fraction of a side of a polygon, and no two matches overlapping or crossing each other. What is the smallest total area of the polygons Mona can make?",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 364,
        "difficulty": 4.0
      },
      {
        "id": "3818fa88e32fdabf525fb9f9a69cc4f0680c3f881160f48b58daa69a25346ad9",
        "domain": "omni",
        "question": "For any positive integer $n$, let \\langle n\\rangle denote the closest integer to \\sqrt{n}.  Evaluate\n\\[\\sum_{n=1}^\\infty \\frac{2^{\\langle n\\rangle}+2^{-\\langle n\\rangle}}{2^n}.\\]",
        "answer": "3",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Multi-variable; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "putnam",
        "source_index": 3221,
        "difficulty": 7.0
      },
      {
        "id": "d8edde385739fe9202455ea1d8391ed5549b11660116dba9bb1bc9a1dd854e7f",
        "domain": "omni",
        "question": "Find the sum of the ages of everyone who wrote a problem for this year's HMMT November contest. If your answer is $X$ and the actual value is $Y$, your score will be $\\max (0,20-|X-Y|)$",
        "answer": "258",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1904,
        "difficulty": 3.5
      },
      {
        "id": "6447068894d11acfc12b33176226b06db06bbfc6cb59c5eb4096b5fba45b0172",
        "domain": "omni",
        "question": "Given that three roots of $f(x) = x^{4} + ax^{2} + bx + c$ are $2, -3$, and $5$, what is the value of $a + b + c$?",
        "answer": "79",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1306,
        "difficulty": 2.5
      },
      {
        "id": "839134dda1de07ce852a1853ee94eac82fd5f02f1733ddf450c6224300802da3",
        "domain": "omni",
        "question": "Vivek is painting three doors numbered 1, 2, and 3. Each door is to be painted either black or gold. How many different ways can the three doors be painted?",
        "answer": "8",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "fermat",
        "source_index": 2863,
        "difficulty": 2.0
      },
      {
        "id": "efb38df46de9c960dce49218c26ae74b13f0478a462f37bd098540224de205b2",
        "domain": "omni",
        "question": "Let $n$ be the answer to this problem. Box $B$ initially contains $n$ balls, and Box $A$ contains half as many balls as Box $B$. After 80 balls are moved from Box $A$ to Box $B$, the ratio of balls in Box $A$ to Box $B$ is now $\\frac{p}{q}$, where $p, q$ are positive integers with $\\operatorname{gcd}(p, q)=1$. Find $100p+q$.",
        "answer": "720",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 2181,
        "difficulty": 4.0
      },
      {
        "id": "4537d25dd7e83a31225c1f30811c12e49c63f3ae03168c2e8a993c843e1d5ae2",
        "domain": "omni",
        "question": "For positive integers $n$, let $L(n)$ be the largest factor of $n$ other than $n$ itself. Determine the number of ordered pairs of composite positive integers $(m, n)$ for which $L(m) L(n)=80$.",
        "answer": "12",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2084,
        "difficulty": 3.5
      },
      {
        "id": "dea855c1058a77ec2ad1a013b2f01e5d8237c27c18e29e9560e77d2e6c3c90ef",
        "domain": "omni",
        "question": "Alex chose positive integers $a, b, c, d, e, f$ and completely multiplied out the polynomial product $(1-x)^{a}(1+x)^{b}\\left(1-x+x^{2}\\right)^{c}\\left(1+x^{2}\\right)^{d}\\left(1+x+x^{2}\\right)^{e}\\left(1+x+x^{2}+x^{3}+x^{4}\\right)^{f}$. After she simplified her result, she discarded any term involving $x$ to any power larger than 6 and was astonished to see that what was left was $1-2 x$. If $a>d+e+f$ and $b>c+d$ and $e>c$, what value of $a$ did she choose?",
        "answer": "23",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 3548,
        "difficulty": 3.0
      },
      {
        "id": "8f9394305f45a92ef42c8f07fa032529543361c8623bf52d4ea8df444dff026c",
        "domain": "omni",
        "question": "Find all positive integers $n$ such that the inequality $$\\left( \\sum\\limits_{i=1}^n a_i^2\\right) \\left(\\sum\\limits_{i=1}^n a_i \\right) -\\sum\\limits_{i=1}^n a_i^3 \\geq 6 \\prod\\limits_{i=1}^n a_i$$ holds for any $n$ positive numbers $a_1, \\dots, a_n$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "austrianpolish_competition",
        "source_index": 3741,
        "difficulty": 6.0
      },
      {
        "id": "03f07639b1a883f046b18d5e851e3d835e39337c540af96761246aba6ff83706",
        "domain": "omni",
        "question": "Let $\\zeta=\\cos \\frac{2 \\pi}{13}+i \\sin \\frac{2 \\pi}{13}$. Suppose $a>b>c>d$ are positive integers satisfying $$\\left|\\zeta^{a}+\\zeta^{b}+\\zeta^{c}+\\zeta^{d}\\right|=\\sqrt{3}$$ Compute the smallest possible value of $1000 a+100 b+10 c+d$.",
        "answer": "7521",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1792,
        "difficulty": 7.0
      },
      {
        "id": "b9aca5d915e94b9a4360d17c1ba3f8dc1819ebaccdc0f97355e4266de0ec3ed2",
        "domain": "omni",
        "question": "Call a set of positive integers good if there is a partition of it into two sets $S$ and $T$, such that there do not exist three elements $a, b, c \\in S$ such that $a^{b}=c$ and such that there do not exist three elements $a, b, c \\in T$ such that $a^{b}=c$ ( $a$ and $b$ need not be distinct). Find the smallest positive integer $n$ such that the set $\\{2,3,4, \\ldots, n\\}$ is not good.",
        "answer": "65536",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 2444,
        "difficulty": 4.5
      },
      {
        "id": "02ac796823cc9beaf6d51a04dc64735958db045e9b544c9922135136b7f1d868",
        "domain": "omni",
        "question": "Michael picks a random subset of the complex numbers \\(\\left\\{1, \\omega, \\omega^{2}, \\ldots, \\omega^{2017}\\right\\}\\) where \\(\\omega\\) is a primitive \\(2018^{\\text {th }}\\) root of unity and all subsets are equally likely to be chosen. If the sum of the elements in his subset is \\(S\\), what is the expected value of \\(|S|^{2}\\)? (The sum of the elements of the empty set is 0.)",
        "answer": "\\frac{1009}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "HMMT_2",
        "source_index": 1555,
        "difficulty": 5.25
      },
      {
        "id": "3727b3a621d726cb6b87f4327bb309346d1aa27f0e58ad1de0256f1167e2d546",
        "domain": "omni",
        "question": "A triangle in the $x y$-plane is such that when projected onto the $x$-axis, $y$-axis, and the line $y=x$, the results are line segments whose endpoints are $(1,0)$ and $(5,0),(0,8)$ and $(0,13)$, and $(5,5)$ and $(7.5,7.5)$, respectively. What is the triangle's area?",
        "answer": "\\frac{17}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 2334,
        "difficulty": 5.25
      },
      {
        "id": "8587a141458b048874a210dfb3503f9bd90c745b8f8df171054539cd67251eb8",
        "domain": "omni",
        "question": "Bob knows that Alice has 2021 secret positive integers $x_{1}, \\ldots, x_{2021}$ that are pairwise relatively prime. Bob would like to figure out Alice's integers. He is allowed to choose a set $S \\subseteq\\{1,2, \\ldots, 2021\\}$ and ask her for the product of $x_{i}$ over $i \\in S$. Alice must answer each of Bob's queries truthfully, and Bob may use Alice's previous answers to decide his next query. Compute the minimum number of queries Bob needs to guarantee that he can figure out each of Alice's integers.",
        "answer": "11",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 853,
        "difficulty": 5.25
      },
      {
        "id": "5ac776179ccfeaa23516b9084f363a1ba3998a58de57f7039c00c81524727b6b",
        "domain": "omni",
        "question": "Each of $a, b$ and $c$ is equal to a number from the list $3^{1}, 3^{2}, 3^{3}, 3^{4}, 3^{5}, 3^{6}, 3^{7}, 3^{8}$. There are $N$ triples $(a, b, c)$ with $a \\leq b \\leq c$ for which each of $\\frac{ab}{c}, \\frac{ac}{b}$ and $\\frac{bc}{a}$ is equal to an integer. What is the value of $N$?",
        "answer": "86",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3181,
        "difficulty": 2.0
      },
      {
        "id": "0026d94e7c82e3bfa9c574f38bc06b1af298b5f8470e31cb22bb8081569012d2",
        "domain": "omni",
        "question": "Suppose $A B C$ is a triangle with incircle $\\omega$, and $\\omega$ is tangent to $\\overline{B C}$ and $\\overline{C A}$ at $D$ and $E$ respectively. The bisectors of $\\angle A$ and $\\angle B$ intersect line $D E$ at $F$ and $G$ respectively, such that $B F=1$ and $F G=G A=6$. Compute the radius of $\\omega$.",
        "answer": "\\frac{2 \\sqrt{5}}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 587,
        "difficulty": 5.25
      },
      {
        "id": "31ee3d90cd7ce8f8de2a95c57826166fec1f5e52837af9050e725f72b1acb77f",
        "domain": "omni",
        "question": "For some positive real $\\alpha$, the set $S$ of positive real numbers $x$ with $\\{x\\}>\\alpha x$ consists of the union of several intervals, with total length 20.2. The value of $\\alpha$ can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$. (Here, $\\{x\\}=x-\\lfloor x\\rfloor$ is the fractional part of $x$.)",
        "answer": "4633",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2505,
        "difficulty": 3.5
      },
      {
        "id": "0283d865cc9953cff36c591b4f06263c88017365aad3d040b4b9947539df2479",
        "domain": "omni",
        "question": "There exist unique nonnegative integers $A, B$ between 0 and 9, inclusive, such that $(1001 \\cdot A+110 \\cdot B)^{2}=57,108,249$. Find $10 \\cdot A+B$.",
        "answer": "75",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2265,
        "difficulty": 4.0
      },
      {
        "id": "54e4ca99a79776a5ec4b3e506cf9cad6e1ce24e204fd334437468491061618c6",
        "domain": "omni",
        "question": "What is the minimum value of the product $\\prod_{i=1}^{6} \\frac{a_{i}-a_{i+1}}{a_{i+2}-a_{i+3}}$ given that $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}, a_{6}\\right)$ is a permutation of $(1,2,3,4,5,6)$?",
        "answer": "1",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations",
        "source": "HMMT_11",
        "source_index": 1824,
        "difficulty": 4.5
      },
      {
        "id": "58b17d986f8105a7ce948f0d2720f8d42ffd6449aff7c320098541820bb0189d",
        "domain": "omni",
        "question": "Carina is in a tournament in which no game can end in a tie. She continues to play games until she loses 2 games, at which point she is eliminated and plays no more games. The probability of Carina winning the first game is $\frac{1}{2}$. After she wins a game, the probability of Carina winning the next game is $\frac{3}{4}$. After she loses a game, the probability of Carina winning the next game is $\frac{1}{3}$. What is the probability that Carina wins 3 games before being eliminated from the tournament?",
        "answer": "23",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "cayley",
        "source_index": 2730,
        "difficulty": 2.5
      },
      {
        "id": "e52a687d60c89c3558e9f06c0aa554a33b260c97953bc8083862dc0189238e95",
        "domain": "omni",
        "question": "A point $P$ lies at the center of square $A B C D$. A sequence of points $\\left\\{P_{n}\\right\\}$ is determined by $P_{0}=P$, and given point $P_{i}$, point $P_{i+1}$ is obtained by reflecting $P_{i}$ over one of the four lines $A B, B C, C D, D A$, chosen uniformly at random and independently for each $i$. What is the probability that $P_{8}=P$ ?",
        "answer": "\\frac{1225}{16384}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1226,
        "difficulty": 5.0
      },
      {
        "id": "47ace0c8ece97dabdf3703de58d597147f1b24de188630def6946507c3244abc",
        "domain": "omni",
        "question": "Find the sum of the coefficients of the polynomial $P(x)=x^{4}-29 x^{3}+a x^{2}+b x+c$, given that $P(5)=11, P(11)=17$, and $P(17)=23$.",
        "answer": "-3193",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2215,
        "difficulty": 4.5
      },
      {
        "id": "a4b3c7c1b322184b638747a525866e865f930d7ab006ec9275c14f294a0251b5",
        "domain": "omni",
        "question": "The sequence $\\{a_n\\}_{n\\geq 0}$ of real numbers satisfies the relation:\n\\[ a_{m+n} + a_{m-n} - m + n -1 = \\frac12 (a_{2m} + a_{2n})  \\]\nfor all non-negative integers $m$ and $n$, $m \\ge n$. If $a_1 = 3$ find $a_{2004}$.",
        "answer": "4018021",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "balkan_mo",
        "source_index": 3862,
        "difficulty": 5.0
      },
      {
        "id": "3ff086c49f382e18633066c3728aa3860cdf9976d962f54535ead3ef298411ba",
        "domain": "omni",
        "question": "2019 points are chosen independently and uniformly at random on the interval $[0,1]$. Tairitsu picks 1000 of them randomly and colors them black, leaving the remaining ones white. Hikari then computes the sum of the positions of the leftmost white point and the rightmost black point. What is the probability that this sum is at most 1 ?",
        "answer": "\\frac{1019}{2019}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 866,
        "difficulty": 4.5
      },
      {
        "id": "09b7ca60c1fc9397572d392da0ebdfbabe8b1b19a9acd3523e40834f722c27d5",
        "domain": "omni",
        "question": "A regular octahedron $A B C D E F$ is given such that $A D, B E$, and $C F$ are perpendicular. Let $G, H$, and $I$ lie on edges $A B, B C$, and $C A$ respectively such that \\frac{A G}{G B}=\\frac{B H}{H C}=\\frac{C I}{I A}=\\rho. For some choice of $\\rho>1, G H, H I$, and $I G$ are three edges of a regular icosahedron, eight of whose faces are inscribed in the faces of $A B C D E F$. Find $\\rho$.",
        "answer": "(1+\\sqrt{5}) / 2",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 997,
        "difficulty": 5.25
      },
      {
        "id": "acefdbc2b1cac25ecde3acc59ff8e5643e68978d68bc816dc8beb38ab15cf582",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. Company XYZ wants to locate their base at the point $P$ in the plane minimizing the total distance to their workers, who are located at vertices $A, B$, and $C$. There are 1,5 , and 4 workers at $A, B$, and $C$, respectively. Find the minimum possible total distance Company XYZ's workers have to travel to get to $P$.",
        "answer": "69",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2049,
        "difficulty": 4.0
      },
      {
        "id": "dff2bafb9f1c16dbe21153a0f82423a16d0baf9931527b4d6c4860de89d1cd29",
        "domain": "omni",
        "question": "A cylinder with radius 15 and height 16 is inscribed in a sphere. Three congruent smaller spheres of radius $x$ are externally tangent to the base of the cylinder, externally tangent to each other, and internally tangent to the large sphere. What is the value of $x$?",
        "answer": "\\frac{15 \\sqrt{37}-75}{4}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2146,
        "difficulty": 4.5
      },
      {
        "id": "229c9428dbec0e45ac6a35b0843273afc2d4c1c421d925678263779a8d92fbdd",
        "domain": "omni",
        "question": "Let $\\otimes$ be a binary operation that takes two positive real numbers and returns a positive real number. Suppose further that $\\otimes$ is continuous, commutative $(a \\otimes b=b \\otimes a)$, distributive across multiplication $(a \\otimes(b c)=(a \\otimes b)(a \\otimes c))$, and that $2 \\otimes 2=4$. Solve the equation $x \\otimes y=x$ for $y$ in terms of $x$ for $x>1$.",
        "answer": "\\sqrt{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 3347,
        "difficulty": 5.0
      },
      {
        "id": "e5c76fbf62501c3aad6fe4845f1e36bf675d82d62c969561d9453727e1e65e49",
        "domain": "omni",
        "question": "A moth starts at vertex $A$ of a certain cube and is trying to get to vertex $B$, which is opposite $A$, in five or fewer \"steps,\" where a step consists in traveling along an edge from one vertex to another. The moth will stop as soon as it reaches $B$. How many ways can the moth achieve its objective?",
        "answer": "48",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1124,
        "difficulty": 4.0
      },
      {
        "id": "ddbd159a643c535637b148aadf0b9b3f13b044a7efc3797164c181388d435b67",
        "domain": "omni",
        "question": "The first four terms of a sequence are $1,4,2$, and 3. Beginning with the fifth term in the sequence, each term is the sum of the previous four terms. What is the eighth term?",
        "answer": "66",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2867,
        "difficulty": 2.0
      },
      {
        "id": "a83914a80c6d1af635ffa5b2d9377a098b83d5946eca0d21c318b50046a60804",
        "domain": "omni",
        "question": "Suppose that $m$ and $n$ are positive integers with $m<n$ such that the interval $[m, n)$ contains more multiples of 2021 than multiples of 2000. Compute the maximum possible value of $n-m$.",
        "answer": "191999",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1402,
        "difficulty": 4.0
      },
      {
        "id": "89b6ca2361bac204fb1a648edcd492968accdcfc0dc4629d5c4aba61ce96b259",
        "domain": "omni",
        "question": "Petya bought one cake, two cupcakes and three bagels, Apya bought three cakes and a bagel, and Kolya bought six cupcakes. They all paid the same amount of money for purchases. Lena bought two cakes and two bagels. And how many cupcakes could be bought for the same amount spent to her?",
        "answer": "$\\frac{13}{4}$",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "caucasus_mathematical_olympiad",
        "source_index": 3991,
        "difficulty": 5.0
      },
      {
        "id": "fa5d48048664bb9f31455d19e28564545fb40b28c14e98c574b3e55c46ff5b7a",
        "domain": "omni",
        "question": "Meg started with the number 100. She increased this number by $20\\%$ and then increased the resulting number by $50\\%$. What was her final result?",
        "answer": "180",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Percentages -> Other; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3516,
        "difficulty": 1.5
      },
      {
        "id": "b20cadf2c4f9cd3dd1a59a7abd5ef618a437b7ce6d5e609234090864e794d056",
        "domain": "omni",
        "question": "Wendy eats sushi for lunch. She wants to eat six pieces of sushi arranged in a $2 \\times 3$ rectangular grid, but sushi is sticky, and Wendy can only eat a piece if it is adjacent to (not counting diagonally) at most two other pieces. In how many orders can Wendy eat the six pieces of sushi, assuming that the pieces of sushi are distinguishable?",
        "answer": "360",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2097,
        "difficulty": 4.0
      },
      {
        "id": "24c67960a29c0d06f8d3c28fee28d4d4ff6f5b2df604a636144f06bca236b124",
        "domain": "omni",
        "question": "The rational numbers $x$ and $y$, when written in lowest terms, have denominators 60 and 70 , respectively. What is the smallest possible denominator of $x+y$ ?",
        "answer": "84",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_2",
        "source_index": 516,
        "difficulty": 3.5
      },
      {
        "id": "488774d6bf842a55f4e911e827ac3348f8061b7c678986c8ca408ef69181850e",
        "domain": "omni",
        "question": "A graph $G(V,E)$ is triangle-free, but adding any edges to the graph will form a triangle. It's given that $|V|=2019$, $|E|>2018$, find the minimum of $|E|$ .",
        "answer": "4033",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "china_team_selection_test",
        "source_index": 274,
        "difficulty": 8.0
      },
      {
        "id": "be1b06c160acedabad6169476a45d01bdd3736d507e0e8f9dc041696141d7d19",
        "domain": "omni",
        "question": "In a single-elimination tournament consisting of $2^{9}=512$ teams, there is a strict ordering on the skill levels of the teams, but Joy does not know that ordering. The teams are randomly put into a bracket and they play out the tournament, with the better team always beating the worse team. Joy is then given the results of all 511 matches and must create a list of teams such that she can guarantee that the third-best team is on the list. What is the minimum possible length of Joy's list?",
        "answer": "45",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2474,
        "difficulty": 5.0
      },
      {
        "id": "9477e06fd2a63d0f13d8a270917f3294a9c466dce0cd6dc79d80d5905eac29de",
        "domain": "omni",
        "question": "Point P_{1} is located 600 miles West of point P_{2}. At 7:00 AM a car departs from P_{1} and drives East at a speed of 50 miles per hour. At 8:00 AM another car departs from P_{2} and drives West at a constant speed of x miles per hour. If the cars meet each other exactly halfway between P_{1} and P_{2}, what is the value of x?",
        "answer": "60",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2058,
        "difficulty": 4.0
      },
      {
        "id": "b4edf5631edeb672066ee94463aa8c063a36ab159dc5046e45c414d9f5019a94",
        "domain": "omni",
        "question": "Circles $C_{1}, C_{2}, C_{3}$ have radius 1 and centers $O, P, Q$ respectively. $C_{1}$ and $C_{2}$ intersect at $A, C_{2}$ and $C_{3}$ intersect at $B, C_{3}$ and $C_{1}$ intersect at $C$, in such a way that $\\angle A P B=60^{\\circ}, \\angle B Q C=36^{\\circ}$, and $\\angle C O A=72^{\\circ}$. Find angle $A B C$ (degrees).",
        "answer": "90",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 960,
        "difficulty": 5.0
      },
      {
        "id": "b190de3bc57ebc6fbdf904bad291dbc5c57536ac2471779eb79475d002e77fc7",
        "domain": "omni",
        "question": "For any positive integer $n, S_{n}$ be the set of all permutations of \\{1,2,3, \\ldots, n\\}. For each permutation $\\pi \\in S_{n}$, let $f(\\pi)$ be the number of ordered pairs $(j, k)$ for which $\\pi(j)>\\pi(k)$ and $1 \\leq j<k \\leq n$. Further define $g(\\pi)$ to be the number of positive integers $k \\leq n$ such that $\\pi(k) \\equiv k \\pm 1(\\bmod n)$. Compute $$\\sum_{\\pi \\in S_{999}}(-1)^{f(\\pi)+g(\\pi)}$$",
        "answer": "995 \\times 2^{998}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Linear Algebra -> Matrices",
        "source": "HMMT_2",
        "source_index": 1389,
        "difficulty": 5.5
      },
      {
        "id": "50165a0ef487b72ee29027fcb4650135eaca4a3bde478bb15302242f295167ce",
        "domain": "omni",
        "question": "Six standard fair six-sided dice are rolled and arranged in a row at random. Compute the expected number of dice showing the same number as the sixth die in the row.",
        "answer": "\\frac{11}{6}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1799,
        "difficulty": 5.0
      },
      {
        "id": "a4012bc7d35dcf209085c7be88137272ed4c1767744ef7e6a48f6664a30ad37e",
        "domain": "omni",
        "question": "If $3n=9+9+9$, what is the value of $n$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2735,
        "difficulty": 1.5
      },
      {
        "id": "11e94869078365769b90462435b442ba06715806cec62308bcc00636e8f871eb",
        "domain": "omni",
        "question": "It takes Pearl 7 days to dig 4 holes. It takes Miguel 3 days to dig 2 holes. If they work together and each continues digging at these same rates, how many holes in total will they dig in 21 days?",
        "answer": "26",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2889,
        "difficulty": 1.5
      },
      {
        "id": "8ad46f5aef792312f335f03a18316fccd565ee60bcf44791dd8678f0bf66ac80",
        "domain": "omni",
        "question": "If $\\frac{1}{3}$ of $x$ is equal to 4, what is $\\frac{1}{6}$ of $x$?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 2983,
        "difficulty": 1.0
      },
      {
        "id": "69c4f9c2aba25e3f2d803397555a509a6083b1be0d38bf8b25c56d3507049433",
        "domain": "omni",
        "question": "Kelvin the Frog and 10 of his relatives are at a party. Every pair of frogs is either friendly or unfriendly. When 3 pairwise friendly frogs meet up, they will gossip about one another and end up in a fight (but stay friendly anyway). When 3 pairwise unfriendly frogs meet up, they will also end up in a fight. In all other cases, common ground is found and there is no fight. If all $\\binom{11}{3}$ triples of frogs meet up exactly once, what is the minimum possible number of fights?",
        "answer": "28",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 500,
        "difficulty": 5.0
      },
      {
        "id": "b7599583cbc98b0799e469c6a9b01097e7d3393dee2537cff7a1e876440441ca",
        "domain": "omni",
        "question": "Three ants begin on three different vertices of a tetrahedron. Every second, they choose one of the three edges connecting to the vertex they are on with equal probability and travel to the other vertex on that edge. They all stop when any two ants reach the same vertex at the same time. What is the probability that all three ants are at the same vertex when they stop?",
        "answer": "\\frac{1}{16}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1773,
        "difficulty": 5.0
      },
      {
        "id": "e37c81f24b1a170cfda5bf999cc7d1543e175d42ab91a49298c8b6a5568b11f0",
        "domain": "omni",
        "question": "Neo has an infinite supply of red pills and blue pills. When he takes a red pill, his weight will double, and when he takes a blue pill, he will lose one pound. If Neo originally weighs one pound, what is the minimum number of pills he must take to make his weight 2015 pounds?",
        "answer": "13",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2103,
        "difficulty": 4.0
      },
      {
        "id": "a16fce16bd2c4e4e8bb6b81295805e55063e8cb5f03fcdd51447f094438b5930",
        "domain": "omni",
        "question": "Say that a polynomial with real coefficients in two variables, $x,y$, is \\emph{balanced} if\nthe average value of the polynomial on each circle centered at the origin is $0$.\nThe balanced polynomials of degree at most $2009$ form a vector space $V$ over $\\mathbb{R}$.\nFind the dimension of $V$.",
        "answer": "2020050",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "putnam",
        "source_index": 3241,
        "difficulty": 8.0
      },
      {
        "id": "c35c87a9d0148b3cf9bddb6573de358bc1f6fd914215fd2aca302d777938371d",
        "domain": "omni",
        "question": "If \\( a=\\frac{2}{3}b \\) and \\( b \\neq 0 \\), what is \\( \\frac{9a+8b}{6a} \\) equal to?",
        "answer": "\\frac{7}{2}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2803,
        "difficulty": 2.5
      },
      {
        "id": "068944125c10e1210202b366aa1564aae0c6d461fffd4ea2ac054899393c31b2",
        "domain": "omni",
        "question": "In how many ways can the numbers $1,2, \\ldots, 2002$ be placed at the vertices of a regular 2002-gon so that no two adjacent numbers differ by more than 2? (Rotations and reflections are considered distinct.)",
        "answer": "4004",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 917,
        "difficulty": 4.0
      },
      {
        "id": "d94495bf35f0c61308412cc6da56b546e9ec6815792d462b6b3b94fe333ac240",
        "domain": "omni",
        "question": "Determine the largest of all integers $n$ with the property that $n$ is divisible by all positive integers that are less than $\\sqrt[3]{n}$.",
        "answer": "420",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "apmoapmo_sol",
        "source_index": 1613,
        "difficulty": 6.0
      },
      {
        "id": "e463d40c83d52404b64b52f366c732b61233d52a61867478a721a5e324c49c29",
        "domain": "omni",
        "question": "In her last basketball game, Jackie scored 36 points. These points raised the average number of points that she scored per game from 20 to 21. To raise this average to 22 points, how many points must Jackie score in her next game?",
        "answer": "38",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3192,
        "difficulty": 1.5
      },
      {
        "id": "cc8eacafe2a423da83439f074a11663defad78f49265f2d214ccedb5a0d3dde2",
        "domain": "omni",
        "question": "How many triples \\((a, b, c)\\) of positive integers satisfy the conditions \\( 6ab = c^2 \\) and \\( a < b < c \\leq 35 \\)?",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "pascal",
        "source_index": 2941,
        "difficulty": 2.5
      },
      {
        "id": "af46c7c9ab3363fb1eb7f8bb82b778daab718bd3f164aebdab489e8da64407f9",
        "domain": "omni",
        "question": "How many integers are greater than $\frac{5}{7}$ and less than $\frac{28}{3}$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2813,
        "difficulty": 2.0
      },
      {
        "id": "32251de5ea0d01bd7c59d8b443550b9a693632d5c44bc45966d362b53dec25a6",
        "domain": "omni",
        "question": "Carl only eats food in the shape of equilateral pentagons. Unfortunately, for dinner he receives a piece of steak in the shape of an equilateral triangle. So that he can eat it, he cuts off two corners with straight cuts to form an equilateral pentagon. The set of possible perimeters of the pentagon he obtains is exactly the interval $[a, b)$, where $a$ and $b$ are positive real numbers. Compute $\\frac{a}{b}$.",
        "answer": "4 \\sqrt{3}-6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2143,
        "difficulty": 5.0
      },
      {
        "id": "e2f464b0ff5b966a486455242982cb225f8db04966c07e4e3c33c6ad84f447e5",
        "domain": "omni",
        "question": "Karim has 23 candies. He eats $n$ candies and divides the remaining candies equally among his three children so that each child gets an integer number of candies. Which of the following is not a possible value of $n$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3414,
        "difficulty": 1.5
      },
      {
        "id": "b913da8cebc28b99e3b94e7a5612f28bb67a9bca51fc4d29418ffd0a77eb661c",
        "domain": "omni",
        "question": "Find the number of triangulations of a general convex 7-gon into 5 triangles by 4 diagonals that do not intersect in their interiors.",
        "answer": "42",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 376,
        "difficulty": 3.5
      },
      {
        "id": "fb8337f9cc3fd3d4329b800641d4140d416f650ef528e2ba1a4fc140e8ebfd5f",
        "domain": "omni",
        "question": "Let $G$ be a simple graph with 100 vertices such that for each vertice $u$, there exists a vertice $v \\in N \\left ( u \\right )$ and $ N \\left ( u \\right ) \\cap  N \\left ( v \\right ) = \\o $. Try to find the maximal possible number of edges in $G$. The $ N \\left ( . \\right )$  refers to the neighborhood.",
        "answer": "3822",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "china_team_selection_test",
        "source_index": 73,
        "difficulty": 9.0
      },
      {
        "id": "864a8af3c878899e7dec9a7f14a1ca360a2fedf1e9f51700ae248a669fce5f96",
        "domain": "omni",
        "question": "Let $k$ be the answer to this problem. The probability that an integer chosen uniformly at random from $\\{1,2, \\ldots, k\\}$ is a multiple of 11 can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "1000",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 935,
        "difficulty": 5.25
      },
      {
        "id": "fb2e0ee95fd8abc8793c412c4ac3882f715fea9fcebe2e71c6b4d79a5b111582",
        "domain": "omni",
        "question": "Compute the number of ordered pairs of integers $(a, b)$, with $2 \\leq a, b \\leq 2021$, that satisfy the equation $$a^{\\log _{b}\\left(a^{-4}\\right)}=b^{\\log _{a}\\left(b a^{-3}\\right)}.$$",
        "answer": "43",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 1413,
        "difficulty": 5.5
      },
      {
        "id": "1e25e8cc01001aaa5bc3cc9b1b75029e4c92ba253c9d5b60ae144336d3110251",
        "domain": "omni",
        "question": "Let $L$ be the number formed by $2022$ digits equal to $1$, that is, $L=1111\\dots 111$.\nCompute the sum of the digits of the number $9L^2+2L$.",
        "answer": "4044",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "all_levels",
        "source_index": 4114,
        "difficulty": 5.0
      },
      {
        "id": "818ca1f45032283e7fb64109ce4e2a6fd56c1e447c10aeb164e465819767b77a",
        "domain": "omni",
        "question": "Let $A$ be a $n\\times n$ matrix such that $A_{ij} = i+j$. Find the rank of $A$.\n\n[hide=\"Remark\"]Not asked in the contest: $A$ is diagonalisable since real symetric matrix it is not difficult to find its eigenvalues.[/hide]",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Matrices; Mathematics -> Algebra -> Linear Algebra -> Determinants",
        "source": "imc",
        "source_index": 3759,
        "difficulty": 7.0
      },
      {
        "id": "d69d2644d295aa84a3fee8a5e945bf2422a28a8f785d6f6db40e43a4e0dcb0a5",
        "domain": "omni",
        "question": "What is the radius of the smallest sphere in which 4 spheres of radius 1 will fit?",
        "answer": "\\frac{2+\\sqrt{6}}{2}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1079,
        "difficulty": 4.5
      },
      {
        "id": "23b2c834b8b7fee4ca74b558d5f772432ce08b95b75e3c2931d11bd59cfd290d",
        "domain": "omni",
        "question": "You have a twig of length 1. You repeatedly do the following: select two points on the twig independently and uniformly at random, make cuts on these two points, and keep only the largest piece. After 2012 repetitions, what is the expected length of the remaining piece?",
        "answer": "\\left(\\frac{11}{18}\\right)^{2012}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 743,
        "difficulty": 5.25
      },
      {
        "id": "bd0a9edc40f27c66665e84a010cfa85ae90d74c0cde0694f700944bc213b1487",
        "domain": "omni",
        "question": "Let $x_{1}, x_{2}, \\ldots, x_{2022}$ be nonzero real numbers. Suppose that $x_{k}+\\frac{1}{x_{k+1}}<0$ for each $1 \\leq k \\leq 2022$, where $x_{2023}=x_{1}$. Compute the maximum possible number of integers $1 \\leq n \\leq 2022$ such that $x_{n}>0$.",
        "answer": "1010",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1708,
        "difficulty": 4.0
      },
      {
        "id": "02c64d0b65c82f87bda21230ae3098db4bd800cefdac3a2cbb9c1e1e1a3c2171",
        "domain": "omni",
        "question": "A point $P$ is chosen uniformly at random inside a square of side length 2. If $P_{1}, P_{2}, P_{3}$, and $P_{4}$ are the reflections of $P$ over each of the four sides of the square, find the expected value of the area of quadrilateral $P_{1} P_{2} P_{3} P_{4}$.",
        "answer": "8",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Areas -> Other",
        "source": "HMMT_11",
        "source_index": 1906,
        "difficulty": 5.0
      },
      {
        "id": "4cc64640173e5c70122e66b91148f0aa0045afb8b967e6051f41342c2648c85f",
        "domain": "omni",
        "question": "What is the perimeter of the triangle formed by the points of tangency of the incircle of a 5-7-8 triangle with its sides?",
        "answer": "\\frac{9 \\sqrt{21}}{7}+3",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2598,
        "difficulty": 4.0
      },
      {
        "id": "a5ca98c69be2f040f0745ef18bd6dc7b72daa60a9c57ab480c86ece66b04fc8b",
        "domain": "omni",
        "question": "For which value of \\( x \\) is \\( x^3 < x^2 \\)?",
        "answer": "\\frac{3}{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "cayley",
        "source_index": 3530,
        "difficulty": 1.5
      },
      {
        "id": "4b9f527ec4ad8a8146df147d9ac9ba56349fd963e69e46be1da1591787dea6df",
        "domain": "omni",
        "question": "Kelvin the Frog is trying to hop across a river. The river has 10 lilypads on it, and he must hop on them in a specific order (the order is unknown to Kelvin). If Kelvin hops to the wrong lilypad at any point, he will be thrown back to the wrong side of the river and will have to start over. Assuming Kelvin is infinitely intelligent, what is the minimum number of hops he will need to guarantee reaching the other side?",
        "answer": "176",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2335,
        "difficulty": 4.5
      },
      {
        "id": "f9f4b24949909f9585cff8f63d92b729c9c88512f3817d4d1b874a262e2eb07f",
        "domain": "omni",
        "question": "In right triangle $A B C$, a point $D$ is on hypotenuse $A C$ such that $B D \\perp A C$. Let $\\omega$ be a circle with center $O$, passing through $C$ and $D$ and tangent to line $A B$ at a point other than $B$. Point $X$ is chosen on $B C$ such that $A X \\perp B O$. If $A B=2$ and $B C=5$, then $B X$ can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "8041",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1372,
        "difficulty": 5.0
      },
      {
        "id": "11e06a9f02255bd5d22affefa7132d576b478b7118ab3a6c445b0eb07ebb4fb0",
        "domain": "omni",
        "question": "(Caos) A cao [sic] has 6 legs, 3 on each side. A walking pattern for the cao is defined as an ordered sequence of raising and lowering each of the legs exactly once (altogether 12 actions), starting and ending with all legs on the ground. The pattern is safe if at any point, he has at least 3 legs on the ground and not all three legs are on the same side. Estimate $N$, the number of safe patterns.",
        "answer": "1416528",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1037,
        "difficulty": 5.0
      },
      {
        "id": "0e9ee91ec96f9e92de0157ae7af26bbf448cb2998a28789fa41b626cbc72938d",
        "domain": "omni",
        "question": "A $10 \\times 10$ table consists of 100 unit cells. A block is a $2 \\times 2$ square consisting of 4 unit cells of the table. A set $C$ of $n$ blocks covers the table (i.e. each cell of the table is covered by some block of $C$ ) but no $n-1$ blocks of $C$ cover the table. Find the largest possible value of n.",
        "answer": "39",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "izho",
        "source_index": 1614,
        "difficulty": 5.5
      },
      {
        "id": "421550b2d3a5a05e43a6cc35f501c99da79f6deb3275791dc4d271ba435d5a13",
        "domain": "omni",
        "question": "Let $A B C D$ be a quadrilateral inscribed in a circle with diameter $\\overline{A D}$. If $A B=5, A C=6$, and $B D=7$, find $C D$.",
        "answer": "\\sqrt{38}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1707,
        "difficulty": 4.0
      },
      {
        "id": "e4ef5ae0be84eea3222bc27dfa920c9c3a1423a912605295dc17f1fad3cc52d1",
        "domain": "omni",
        "question": "$P$ is a point inside triangle $A B C$, and lines $A P, B P, C P$ intersect the opposite sides $B C, C A, A B$ in points $D, E, F$, respectively. It is given that $\\angle A P B=90^{\\circ}$, and that $A C=B C$ and $A B=B D$. We also know that $B F=1$, and that $B C=999$. Find $A F$.",
        "answer": "499 / 500",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 675,
        "difficulty": 3.5
      },
      {
        "id": "54cb9cde8028128f503714326678b8c60fe47f4f425d822021e3abe61dcbf5bc",
        "domain": "omni",
        "question": "In three-dimensional space, let $S$ be the region of points $(x, y, z)$ satisfying $-1 \\leq z \\leq 1$. Let $S_{1}, S_{2}, \\ldots, S_{2022}$ be 2022 independent random rotations of $S$ about the origin ( $0,0,0$). The expected volume of the region $S_{1} \\cap S_{2} \\cap \\cdots \\cap S_{2022}$ can be expressed as $\\frac{a \\pi}{b}$, for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "271619",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1497,
        "difficulty": 5.25
      },
      {
        "id": "b113ae5e09ab09fb03f9d07192146a732db67d7715bb7e8e415a3c2f1514b2ed",
        "domain": "omni",
        "question": "Farmer James invents a new currency, such that for every positive integer $n \\leq 6$, there exists an $n$-coin worth $n$ ! cents. Furthermore, he has exactly $n$ copies of each $n$-coin. An integer $k$ is said to be nice if Farmer James can make $k$ cents using at least one copy of each type of coin. How many positive integers less than 2018 are nice?",
        "answer": "210",
        "topic": "Mathematics -> Algebra -> Other; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2308,
        "difficulty": 4.0
      },
      {
        "id": "42fdedbb26645af6f6d0cad5ae3a3245519470a99fc9bbebc9284b6b1f51be17",
        "domain": "omni",
        "question": "For a positive integer $n$, and a non empty subset $A$ of $\\{1,2,...,2n\\}$, call $A$ good if the set $\\{u\\pm v|u,v\\in A\\}$ does not contain the set $\\{1,2,...,n\\}$. Find the smallest real number $c$, such that for any positive integer $n$, and any good subset $A$ of  $\\{1,2,...,2n\\}$, $|A|\\leq cn$.",
        "answer": "\\frac{6}{5}",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 135,
        "difficulty": 8.0
      },
      {
        "id": "0fb6abf4ce93d978904a17009a667e5b623dfffe8d38279141a8283d1ee68ce5",
        "domain": "omni",
        "question": "Four points, $A, B, C$, and $D$, are chosen randomly on the circumference of a circle with independent uniform probability. What is the expected number of sides of triangle $A B C$ for which the projection of $D$ onto the line containing the side lies between the two vertices?",
        "answer": "3/2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1496,
        "difficulty": 3.5
      },
      {
        "id": "9e49ed9a50983bdbc9f5306c5b0019efa147b3b9077872b578b9bf5771295666",
        "domain": "omni",
        "question": "Three points are chosen inside a unit cube uniformly and independently at random. What is the probability that there exists a cube with side length $\\frac{1}{2}$ and edges parallel to those of the unit cube that contains all three points?",
        "answer": "\\frac{1}{8}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 959,
        "difficulty": 4.0
      },
      {
        "id": "5e598c3399c56f266405efc2e4213c9491f55c2dbec81335aee7e4fd385e06b4",
        "domain": "omni",
        "question": "Let $p$ be a real number between 0 and 1. Jocelin has a coin that lands heads with probability $p$ and tails with probability $1-p$; she also has a number written on a blackboard. Each minute, she flips the coin, and if it lands heads, she replaces the number $x$ on the blackboard with $3 x+1$; if it lands tails she replaces it with $x / 2$. Given that there are constants $a, b$ such that the expected value of the value written on the blackboard after $t$ minutes can be written as $a t+b$ for all positive integers $t$, compute $p$.",
        "answer": "\\frac{1}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1993,
        "difficulty": 5.25
      },
      {
        "id": "b27f4ea2d69d31c7ce5ec9515b297c8f498baae0d573379d2e648e75a48770f0",
        "domain": "omni",
        "question": "What is the value of $n$ if $2^{n}=8^{20}$?",
        "answer": "60",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3115,
        "difficulty": 2.0
      },
      {
        "id": "7076e1bee8de5efed51a04a3475a8d64c843c85515d095a472dc5e938df237f9",
        "domain": "omni",
        "question": "Given right triangle $ABC$, with $AB=4, BC=3$, and $CA=5$. Circle $\\omega$ passes through $A$ and is tangent to $BC$ at $C$. What is the radius of $\\omega$?",
        "answer": "\\frac{25}{8}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 549,
        "difficulty": 4.5
      },
      {
        "id": "e716f72fb748ceb1d4cf699dbef6c6a36fb4aac2bff0201b4ff1a739f652ddb8",
        "domain": "omni",
        "question": "Let $A_{12}$ denote the answer to problem 12. There exists a unique triple of digits $(B, C, D)$ such that $10>A_{12}>B>C>D>0$ and $$\\overline{A_{12} B C D}-\\overline{D C B A_{12}}=\\overline{B D A_{12} C}$$ where $\\overline{A_{12} B C D}$ denotes the four digit base 10 integer. Compute $B+C+D$.",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "HMMT_2",
        "source_index": 1512,
        "difficulty": 4.5
      },
      {
        "id": "6daf76230d6cf9a3221d379632d4a64375f408f4ff83bdc965ff60870af1f26e",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, \\ldots, a_{2005}$ be real numbers such that $$\\begin{array}{ccccccccccc} a_{1} \\cdot 1 & + & a_{2} \\cdot 2 & + & a_{3} \\cdot 3 & + & \\cdots & + & a_{2005} \\cdot 2005 & = & 0 \\\\ a_{1} \\cdot 1^{2} & + & a_{2} \\cdot 2^{2} & + & a_{3} \\cdot 3^{2} & + & \\cdots & + & a_{2005} \\cdot 2005^{2} & = & 0 \\\\ a_{1} \\cdot 1^{3} & + & a_{2} \\cdot 2^{3} & + & a_{3} \\cdot 3^{3} & + & \\cdots & + & a_{2005} \\cdot 2005^{3} & = & 0 \\\\ \\vdots & & \\vdots & & \\vdots & & & & \\vdots & & \\vdots \\\\ a_{1} \\cdot 1^{2004} & + & a_{2} \\cdot 2^{2004} & + & a_{3} \\cdot 3^{2004} & + & \\cdots & + & a_{2005} \\cdot 2005^{2004} & = & 0 \\end{array}$$ and $$a_{1} \\cdot 1^{2005}+a_{2} \\cdot 2^{2005}+a_{3} \\cdot 3^{2005}+\\cdots+a_{2005} \\cdot 2005^{2005}=1$$ What is the value of $a_{1}$?",
        "answer": "1 / 2004!",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1197,
        "difficulty": 4.0
      },
      {
        "id": "e5e9644cc708acdd93632298040036a72e35ae6be4f1115ef311a08b800dd3c7",
        "domain": "omni",
        "question": "If $2 x^{2}=9 x-4$ and $x \neq 4$, what is the value of $2 x$?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 3210,
        "difficulty": 2.0
      },
      {
        "id": "e3c4c26423b77e5b5764b285b02b65c77b7550ac4048e6d118dbb6191b18e6c0",
        "domain": "omni",
        "question": "We wish to color the integers $1,2,3, \\ldots, 10$ in red, green, and blue, so that no two numbers $a$ and $b$, with $a-b$ odd, have the same color. (We do not require that all three colors be used.) In how many ways can this be done?",
        "answer": "186",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1433,
        "difficulty": 5.0
      },
      {
        "id": "10b4da7cf4a2e8210cf137e5935777ba150aa21d4c92621cb08ddb75899a9c27",
        "domain": "omni",
        "question": "Consider a three-person game involving the following three types of fair six-sided dice. - Dice of type $A$ have faces labelled $2,2,4,4,9,9$. - Dice of type $B$ have faces labelled $1,1,6,6,8,8$. - Dice of type $C$ have faces labelled $3,3,5,5,7,7$. All three players simultaneously choose a die (more than one person can choose the same type of die, and the players don't know one another's choices) and roll it. Then the score of a player $P$ is the number of players whose roll is less than $P$ 's roll (and hence is either 0,1 , or 2 ). Assuming all three players play optimally, what is the expected score of a particular player?",
        "answer": "\\frac{8}{9}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1007,
        "difficulty": 5.25
      },
      {
        "id": "e1e38338cb35cc25565380df9fe312d81a84cd1ab4c3d64e6a736a6ee6550f38",
        "domain": "omni",
        "question": "How many ways are there to place four points in the plane such that the set of pairwise distances between the points consists of exactly 2 elements? (Two configurations are the same if one can be obtained from the other via rotation and scaling.)",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_11",
        "source_index": 1768,
        "difficulty": 5.0
      },
      {
        "id": "bd1a3b83b2baa8c552846468c0bc8403a6c58293e3779d7281dbfbb97edfb6ec",
        "domain": "omni",
        "question": "Find the sum of the infinite series $\\frac{1}{3^{2}-1^{2}}\\left(\\frac{1}{1^{2}}-\\frac{1}{3^{2}}\\right)+\\frac{1}{5^{2}-3^{2}}\\left(\\frac{1}{3^{2}}-\\frac{1}{5^{2}}\\right)+\\frac{1}{7^{2}-5^{2}}\\left(\\frac{1}{5^{2}}-\\frac{1}{7^{2}}\\right)+$",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 449,
        "difficulty": 4.5
      },
      {
        "id": "fe455ea697ec21bb5e223a1e1cdc1a8aebd3556b47320d2b8e2b22d89ff6ece1",
        "domain": "omni",
        "question": "Wendy is playing darts with a circular dartboard of radius 20. Whenever she throws a dart, it lands uniformly at random on the dartboard. At the start of her game, there are 2020 darts placed randomly on the board. Every turn, she takes the dart farthest from the center, and throws it at the board again. What is the expected number of darts she has to throw before all the darts are within 10 units of the center?",
        "answer": "6060",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2482,
        "difficulty": 5.25
      },
      {
        "id": "fcd5164cc9d3c888400a6b7c08452a2e1b4030ae93c73e577b957c4f95222ce8",
        "domain": "omni",
        "question": "In the $3 \times 3$ grid shown, the central square contains the integer 5. The remaining eight squares contain $a, b, c, d, e, f, g, h$, which are each to be replaced with an integer from 1 to 9, inclusive. Integers can be repeated. There are $N$ ways to complete the grid so that the sums of the integers along each row, along each column, and along the two main diagonals are all divisible by 5. What are the rightmost two digits of $N$?",
        "answer": "73",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "fermat",
        "source_index": 2853,
        "difficulty": 2.5
      },
      {
        "id": "d1634a4afac35ce444782d2b300d3668701b1d2101dc47b9c9e350be20a0135f",
        "domain": "omni",
        "question": "Suppose there exists a convex $n$-gon such that each of its angle measures, in degrees, is an odd prime number. Compute the difference between the largest and smallest possible values of $n$.",
        "answer": "356",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 996,
        "difficulty": 5.0
      },
      {
        "id": "88e47c81ec568a80cfa4dbcdabfcdc06e871cfd5a61cc190b0b4b4c147cc796b",
        "domain": "omni",
        "question": "20 players are playing in a Super Smash Bros. Melee tournament. They are ranked $1-20$, and player $n$ will always beat player $m$ if $n<m$. Out of all possible tournaments where each player plays 18 distinct other players exactly once, one is chosen uniformly at random. Find the expected number of pairs of players that win the same number of games.",
        "answer": "4",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2228,
        "difficulty": 4.0
      },
      {
        "id": "6011ef8843f6c45b000e3d53932ba2b5d988a2bb59d5457eea82a6f31b5f0e3d",
        "domain": "omni",
        "question": "Let $W, S$ be as in problem 32. Let $A$ be the least positive integer such that an acute triangle with side lengths $S, A$, and $W$ exists. Find $A$.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 608,
        "difficulty": 4.5
      },
      {
        "id": "0c3584181fe84c5e99868e730a72ee8bf020c3cc82dc20bbfb1332c370fa18b8",
        "domain": "omni",
        "question": "Consider a sequence $\\{a_n\\}$ of integers, satisfying $a_1=1, a_2=2$ and $a_{n+1}$ is the largest prime divisor of $a_1+a_2+\\ldots+a_n$. Find $a_{100}$.",
        "answer": "53",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "cono_sur_olympiad",
        "source_index": 3603,
        "difficulty": 5.0
      },
      {
        "id": "7fd5c0008f2d75479c7c3878ebb17b1bb77f3582fb1b8e1cfda20e4b35bfa506",
        "domain": "omni",
        "question": "Which of the following fractions has the greatest value: $\\frac{3}{10}$, $\\frac{4}{7}$, $\\frac{5}{23}$, $\\frac{2}{3}$, $\\frac{1}{2}$?",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3455,
        "difficulty": 2.0
      },
      {
        "id": "5bf6021f58c30f3b9c4786a3fb01a1f04c220dbc174ab9309510674c5f4cef76",
        "domain": "omni",
        "question": "How many different-looking arrangements are possible when four balls are selected at random from six identical red balls and three identical green balls and then arranged in a line?",
        "answer": "15",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "fermat",
        "source_index": 2761,
        "difficulty": 2.5
      },
      {
        "id": "04f0560c15cba75c35f912effebe8c74359e5754fe8bb32923acb745c4a75a49",
        "domain": "omni",
        "question": "Compute the number of triples $(f, g, h)$ of permutations on $\\{1,2,3,4,5\\}$ such that $$ \\begin{aligned} & f(g(h(x)))=h(g(f(x)))=g(x), \\\\ & g(h(f(x)))=f(h(g(x)))=h(x), \\text { and } \\\\ & h(f(g(x)))=g(f(h(x)))=f(x) \\end{aligned} $$ for all $x \\in\\{1,2,3,4,5\\}$.",
        "answer": "146",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1151,
        "difficulty": 4.5
      },
      {
        "id": "77c267347aee0d7e347eec84df5e424372868f55d4b1fc8bdc1823861df5f749",
        "domain": "omni",
        "question": "Let $a=256$. Find the unique real number $x>a^{2}$ such that $$\\log _{a} \\log _{a} \\log _{a} x=\\log _{a^{2}} \\log _{a^{2}} \\log _{a^{2}} x$$",
        "answer": "2^{32}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 1441,
        "difficulty": 4.0
      },
      {
        "id": "abfcb4ec2cad330f8324d01a450090cbf0a346fc6ff2fcaae3cbf615ccbb5aae",
        "domain": "omni",
        "question": "Alice is bored in class, so she thinks of a positive integer. Every second after that, she subtracts from her current number its smallest prime divisor, possibly itself. After 2022 seconds, she realizes that her number is prime. Find the sum of all possible values of her initial number.",
        "answer": "8093",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2175,
        "difficulty": 5.0
      },
      {
        "id": "a4b229e1999e2b85f3cef952e87160ac9bf291ff4b676eae1d9145ef7ec7775a",
        "domain": "omni",
        "question": "How many polynomials of degree exactly 5 with real coefficients send the set \\{1,2,3,4,5,6\\} to a permutation of itself?",
        "answer": "714",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1326,
        "difficulty": 5.0
      },
      {
        "id": "7f7488d67a8864374d78d6894c1d11bde61b273d5a0463c639251df87253895e",
        "domain": "omni",
        "question": "If \\( 50\\% \\) of \\( N \\) is 16, what is \\( 75\\% \\) of \\( N \\)?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 2871,
        "difficulty": 1.5
      },
      {
        "id": "a0585fe8e2cccf1c8de3768962c287dbfad40ca24073224a47de87f4a02e78e1",
        "domain": "omni",
        "question": "If $3^{x}=5$, what is the value of $3^{x+2}$?",
        "answer": "45",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "fermat",
        "source_index": 3044,
        "difficulty": 2.5
      },
      {
        "id": "d34fba17c1ba04998dc09e0fb15432335dac40f2bd11ee2c3de1b837e78be21d",
        "domain": "omni",
        "question": "There are 100 positive integers written on a board. At each step, Alex composes 50 fractions using each number written on the board exactly once, brings these fractions to their irreducible form, and then replaces the 100 numbers on the board with the new numerators and denominators to create 100 new numbers. \n\nFind the smallest positive integer $n{}$ such that regardless of the values of the initial 100 numbers, after $n{}$ steps Alex can arrange to have on the board only pairwise coprime numbers.",
        "answer": "99",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Factorization",
        "source": "balkan_mo_shortlist",
        "source_index": 3607,
        "difficulty": 6.5
      },
      {
        "id": "0205a9847ea0d121e5648c23e7f41ec6b8971b7e2385407cdd92e952b4efbe05",
        "domain": "omni",
        "question": "Consider positive integers $a \\leq b \\leq c \\leq d \\leq e$. There are $N$ lists $a, b, c, d, e$ with a mean of 2023 and a median of 2023, in which the integer 2023 appears more than once, and in which no other integer appears more than once. What is the sum of the digits of $N$?",
        "answer": "28",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3045,
        "difficulty": 2.0
      },
      {
        "id": "a3d254237f2455bc5b7c65823b4d073fceba00cfc8e9beedc443a1238fa7a2b4",
        "domain": "omni",
        "question": "Let $a_{0}=-2, b_{0}=1$, and for $n \\geq 0$, let $$\\begin{aligned} & a_{n+1}=a_{n}+b_{n}+\\sqrt{a_{n}^{2}+b_{n}^{2}} \\\\ & b_{n+1}=a_{n}+b_{n}-\\sqrt{a_{n}^{2}+b_{n}^{2}} \\end{aligned}$$ Find $a_{2012}$.",
        "answer": "2^{1006} \\sqrt{2^{2010}+2}-2^{2011}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1354,
        "difficulty": 4.5
      },
      {
        "id": "98c697668e3c748e1a4ff365a6a2bf4a7ad931833d6d981185814a6274f785b4",
        "domain": "omni",
        "question": "$101$ people,  sitting at a round table in any order,  had  $1,2,... , 101$ cards,  respectively.  \nA transfer is someone give  one  card  to one of  the two people  adjacent to him.\nFind the  smallest positive integer $k$ such that   there always  can  through  no more than  $ k $  times transfer,  each person  hold cards of the same number, regardless of  the sitting order.",
        "answer": "42925",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "china_team_selection_test",
        "source_index": 71,
        "difficulty": 7.5
      },
      {
        "id": "59153ef738ece34d65193d5928f06bfc1d8affdfd8221bcb3737f04426661655",
        "domain": "omni",
        "question": "The numbers $4x, 2x-3, 4x-3$ are three consecutive terms in an arithmetic sequence. What is the value of $x$?",
        "answer": "-\\frac{3}{4}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2732,
        "difficulty": 2.5
      },
      {
        "id": "ffd251f78028b5c4f1b922fa86fcd38d6a53f82c28e496875038db2896a536fd",
        "domain": "omni",
        "question": "Find the smallest real constant $\\alpha$ such that for all positive integers $n$ and real numbers $0=y_{0}<$ $y_{1}<\\cdots<y_{n}$, the following inequality holds: $\\alpha \\sum_{k=1}^{n} \\frac{(k+1)^{3 / 2}}{\\sqrt{y_{k}^{2}-y_{k-1}^{2}}} \\geq \\sum_{k=1}^{n} \\frac{k^{2}+3 k+3}{y_{k}}$.",
        "answer": "\\frac{16 \\sqrt{2}}{9}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "HMMT_2",
        "source_index": 600,
        "difficulty": 5.0
      },
      {
        "id": "015c15b1ab0cae967709443a687dddc3ac7f4f1f41e69f6b535a37bbba79a7d9",
        "domain": "omni",
        "question": "Compute $\\frac{\\tan ^{2}\\left(20^{\\circ}\\right)-\\sin ^{2}\\left(20^{\\circ}\\right)}{\\tan ^{2}\\left(20^{\\circ}\\right) \\sin ^{2}\\left(20^{\\circ}\\right)}$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1472,
        "difficulty": 5.0
      },
      {
        "id": "3a4fb8640011ef25fedcaffc34d95f25dec5b4e3c0ae1d51aac4017021e4270d",
        "domain": "omni",
        "question": "Find the value of $$\\sum_{1 \\leq a<b<c} \\frac{1}{2^{a} 3^{b} 5^{c}}$$ (i.e. the sum of $\\frac{1}{2^{a} 3^{b} 5^{c}}$ over all triples of positive integers $(a, b, c)$ satisfying $a<b<c$).",
        "answer": "\\frac{1}{1624}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1336,
        "difficulty": 5.0
      },
      {
        "id": "aef1e863025db864292c820beea85b1062d354e302acca5cf3c26aeb8a87f3a4",
        "domain": "omni",
        "question": "How many four-digit numbers are there in which at least one digit occurs more than once?",
        "answer": "4464",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 903,
        "difficulty": 4.0
      },
      {
        "id": "edc143b7875633fb370be822aec69a0daaba5e48c36910c91fb3e73b09847315",
        "domain": "omni",
        "question": "Circle $\\Omega$ has radius 5. Points $A$ and $B$ lie on $\\Omega$ such that chord $A B$ has length 6. A unit circle $\\omega$ is tangent to chord $A B$ at point $T$. Given that $\\omega$ is also internally tangent to $\\Omega$, find $A T \\cdot B T$.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 1494,
        "difficulty": 4.5
      },
      {
        "id": "17141e584705745d1a02c316a69171fb47942845050399c22ef47a44225bb9c5",
        "domain": "omni",
        "question": "In $\\triangle ABC, D$ and $E$ are the midpoints of $BC$ and $CA$, respectively. $AD$ and $BE$ intersect at $G$. Given that $GEC$D is cyclic, $AB=41$, and $AC=31$, compute $BC$.",
        "answer": "49",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2473,
        "difficulty": 4.0
      },
      {
        "id": "6c1928913658ea35d11d84526f19d82e0220962fa8c91a284678c0acab14a86e",
        "domain": "omni",
        "question": "Consider a $2 \\times 2$ grid of squares. David writes a positive integer in each of the squares. Next to each row, he writes the product of the numbers in the row, and next to each column, he writes the product of the numbers in each column. If the sum of the eight numbers he writes down is 2015, what is the minimum possible sum of the four numbers he writes in the grid?",
        "answer": "88",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2073,
        "difficulty": 4.0
      },
      {
        "id": "30df29ad6ecd60df75d311f3879bfb763f5f2ccb9cf06a056be4058b6967f3f0",
        "domain": "omni",
        "question": "$A B C D E$ is a cyclic convex pentagon, and $A C=B D=C E . A C$ and $B D$ intersect at $X$, and $B D$ and $C E$ intersect at $Y$. If $A X=6, X Y=4$, and $Y E=7$, then the area of pentagon $A B C D E$ can be written as $\\frac{a \\sqrt{b}}{c}$, where $a, b, c$ are integers, $c$ is positive, $b$ is square-free, and $\\operatorname{gcd}(a, c)=1$. Find $100 a+10 b+c$.",
        "answer": "2852",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2616,
        "difficulty": 4.0
      },
      {
        "id": "8d426a113af7f15edd3bae66a62e9a00f2913ec5b078ccc745789cee7fa9ced6",
        "domain": "omni",
        "question": "Let x; y; z be real numbers, satisfying the relations\n$x \\ge 20$\n$y \u0015\\ge 40$\n$z \u0015\\ge 1675$\nx + y + z = 2015\nFind the greatest value of the product P = $x\u0001y z$",
        "answer": "\\frac{721480000}{27}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "jbmo_shortlist",
        "source_index": 4088,
        "difficulty": 5.5
      },
      {
        "id": "bdffbc22403d9b57f487c6506413855ddb4560d8fc08f7c9f8ed74386e25d176",
        "domain": "omni",
        "question": "A group of friends are sharing a bag of candy. On the first day, they eat $\frac{1}{2}$ of the candies in the bag. On the second day, they eat $\frac{2}{3}$ of the remaining candies. On the third day, they eat $\frac{3}{4}$ of the remaining candies. On the fourth day, they eat $\frac{4}{5}$ of the remaining candies. On the fifth day, they eat $\frac{5}{6}$ of the remaining candies. At the end of the fifth day, there is 1 candy remaining in the bag. How many candies were in the bag before the first day?",
        "answer": "720",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3033,
        "difficulty": 1.5
      },
      {
        "id": "82ef17cb3b06360f5a2c5e02605a4241bb48d8a230dfc77c7f53b53a5572deb9",
        "domain": "omni",
        "question": "Let $a, b$, and $c$ be the 3 roots of $x^{3}-x+1=0$. Find $\\frac{1}{a+1}+\\frac{1}{b+1}+\\frac{1}{c+1}$.",
        "answer": "-2",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1206,
        "difficulty": 5.5
      },
      {
        "id": "8d0166f2f19412fdd699b9cffefbc9bf5821c370fad758bb9c04c1c4276514b4",
        "domain": "omni",
        "question": "Let $N=30^{2015}$. Find the number of ordered 4-tuples of integers $(A, B, C, D) \\in\\{1,2, \\ldots, N\\}^{4}$ (not necessarily distinct) such that for every integer $n, A n^{3}+B n^{2}+2 C n+D$ is divisible by $N$.",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1318,
        "difficulty": 5.0
      },
      {
        "id": "ff28ee59d605c9636c1d3ca382dc8897d128f5d3e149ee9d0d65bc7fe4d05c64",
        "domain": "omni",
        "question": "An 8 by 8 grid of numbers obeys the following pattern: 1) The first row and first column consist of all 1s. 2) The entry in the $i$th row and $j$th column equals the sum of the numbers in the $(i-1)$ by $(j-1)$ sub-grid with row less than $i$ and column less than $j$. What is the number in the 8th row and 8th column?",
        "answer": "2508",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 1845,
        "difficulty": 5.0
      },
      {
        "id": "2abdd57996901eba8afe58450a1ef070a99b802817c20e4a682770cbcc84d1ed",
        "domain": "omni",
        "question": "In a survey, 100 students were asked if they like lentils and were also asked if they like chickpeas. A total of 68 students like lentils. A total of 53 like chickpeas. A total of 6 like neither lentils nor chickpeas. How many of the 100 students like both lentils and chickpeas?",
        "answer": "27",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3226,
        "difficulty": 1.5
      },
      {
        "id": "fef8b842b79f1746f8cfcfac0c3b9b152dd57ce83a2b88ae763eee04e18a643c",
        "domain": "omni",
        "question": "What is the largest possible value for $n$ if the average of the two positive integers $m$ and $n$ is 5?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3215,
        "difficulty": 1.5
      },
      {
        "id": "c6c051dc41883dd580d79d6c8215c5c3343f2292536b707e98162c5289bf3865",
        "domain": "omni",
        "question": "We call an even positive integer $n$ [i]nice[/i] if the set $\\{1, 2, \\dots, n\\}$ can be partitioned into $\\frac{n}{2}$ two-element subsets, such that the sum of the elements in each subset is a power of $3$. For example, $6$ is nice, because the set $\\{1, 2, 3, 4, 5, 6\\}$ can be partitioned into subsets $\\{1, 2\\}$, $\\{3, 6\\}$, $\\{4, 5\\}$. Find the number of nice positive integers which are smaller than $3^{2022}$.",
        "answer": "2^{2022} - 1",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "junior_balkan_mo",
        "source_index": 3785,
        "difficulty": 5.5
      },
      {
        "id": "74a76bf9b7686cccf972ebe51e2b1dc374e7204914bc5ae34e03b786bb202502",
        "domain": "omni",
        "question": "Suppose $m>n>1$ are positive integers such that there exist $n$ complex numbers $x_{1}, x_{2}, \\ldots, x_{n}$ for which - $x_{1}^{k}+x_{2}^{k}+\\cdots+x_{n}^{k}=1$ for $k=1,2, \\ldots, n-1$ - $x_{1}^{n}+x_{2}^{n}+\\cdots+x_{n}^{n}=2$; and - $x_{1}^{m}+x_{2}^{m}+\\cdots+x_{n}^{m}=4$. Compute the smallest possible value of $m+n$.",
        "answer": "34",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1515,
        "difficulty": 5.25
      },
      {
        "id": "6a705c62d74ad347698a69e921c4df39118c3491592a82a752f6adb79ed2c8f8",
        "domain": "omni",
        "question": "Let $A_{1}, A_{2}, A_{3}$ be three points in the plane, and for convenience, let $A_{4}=A_{1}, A_{5}=A_{2}$. For $n=1,2$, and 3, suppose that $B_{n}$ is the midpoint of $A_{n} A_{n+1}$, and suppose that $C_{n}$ is the midpoint of $A_{n} B_{n}$. Suppose that $A_{n} C_{n+1}$ and $B_{n} A_{n+2}$ meet at $D_{n}$, and that $A_{n} B_{n+1}$ and $C_{n} A_{n+2}$ meet at $E_{n}$. Calculate the ratio of the area of triangle $D_{1} D_{2} D_{3}$ to the area of triangle $E_{1} E_{2} E_{3}$.",
        "answer": "\\frac{25}{49}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "apmoapmo_sol",
        "source_index": 1460,
        "difficulty": 6.0
      },
      {
        "id": "e82abb73a98e486cf07811d784539bad07ffb41232752479c14e1c4a1d8661b2",
        "domain": "omni",
        "question": "Given any positive integer, we can write the integer in base 12 and add together the digits of its base 12 representation. We perform this operation on the number $7^{6^{5^{3^{2^{1}}}}}$ repeatedly until a single base 12 digit remains. Find this digit.",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2291,
        "difficulty": 4.5
      },
      {
        "id": "4b078b43a80aca1b856c3d9c8c51ddb219fd6d4858810aac6ad82c03727e06e6",
        "domain": "omni",
        "question": "At the Lacsap Hospital, Emily is a doctor and Robert is a nurse. Not including Emily, there are five doctors and three nurses at the hospital. Not including Robert, there are $d$ doctors and $n$ nurses at the hospital. What is the product of $d$ and $n$?",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2891,
        "difficulty": 1.5
      },
      {
        "id": "4af6299ebdeb0b4b571d4b139dec2f6857a6606320787b303ac8ee54f52c56cf",
        "domain": "omni",
        "question": "There are $n$ students in the math club. When grouped in 4s, there is one incomplete group. When grouped in 3s, there are 3 more complete groups than with 4s, and one incomplete group. When grouped in 2s, there are 5 more complete groups than with 3s, and one incomplete group. What is the sum of the digits of $n^{2}-n$?",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3105,
        "difficulty": 2.5
      },
      {
        "id": "d1ba98519fbaaa327789aebe43aa0f51e149bc1215b910ad5de580415c48bc60",
        "domain": "omni",
        "question": "Let $A$ be the area of the largest semicircle that can be inscribed in a quarter-circle of radius 1. Compute $\\frac{120 A}{\\pi}$.",
        "answer": "20",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 2021,
        "difficulty": 4.5
      },
      {
        "id": "1e7dfe346e2af62133d556658a2fba62f369901e5d519ab9c181124073c928b0",
        "domain": "omni",
        "question": "What is the sum of the positive divisors of 1184?",
        "answer": "2394",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2882,
        "difficulty": 2.0
      },
      {
        "id": "dfe4482ef618b66b2970bc86703adcb1ea07469e57cfe3bb4888b0660bac2885",
        "domain": "omni",
        "question": "A cube has side length 1. Find the product of the lengths of the diagonals of this cube (a diagonal is a line between two vertices that is not an edge).",
        "answer": "576",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 1706,
        "difficulty": 3.5
      },
      {
        "id": "00f789edf591674ef37f2816c7b5dcf512648f17ba7033bfffc5a18ae8867b82",
        "domain": "omni",
        "question": "How many ordered triples of positive integers $(a, b, c)$ are there for which $a^{4} b^{2} c=54000$ ?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2026,
        "difficulty": 3.5
      },
      {
        "id": "4e70a29d2ca08618d46f55cc92d09437913eefa8545af99a9d9de95e9210f921",
        "domain": "omni",
        "question": "Kevin has four red marbles and eight blue marbles. He arranges these twelve marbles randomly, in a ring. Determine the probability that no two red marbles are adjacent.",
        "answer": "\\frac{7}{33}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1291,
        "difficulty": 4.0
      },
      {
        "id": "a7b03bde9ec9e49be997d2347d0fde5c5a8f87eb0a0d3d32c057c9506fbb4b23",
        "domain": "omni",
        "question": "Let $f(x)=x^{2}+2 x+1$. Let $g(x)=f(f(\\cdots f(x)))$, where there are $2009 f \\mathrm{~s}$ in the expression for $g(x)$. Then $g(x)$ can be written as $g(x)=x^{2^{2009}}+a_{2^{2009}-1} x^{2^{2009}-1}+\\cdots+a_{1} x+a_{0}$ where the $a_{i}$ are constants. Compute $a_{2^{2009}-1}$.",
        "answer": "2^{2009}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 1688,
        "difficulty": 5.25
      },
      {
        "id": "50a9503485d1d762c32d0931ecef8308e6e4afdf02036c00f640b83eb769176e",
        "domain": "omni",
        "question": "Let $A B C D$ be a unit square. A circle with radius $\\frac{32}{49}$ passes through point $D$ and is tangent to side $A B$ at point $E$. Then $D E=\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "807",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2620,
        "difficulty": 4.0
      },
      {
        "id": "83c76acb2a5dd2bbe0cbc5a167cd08a7fb2b33b98dd328b5a51abc25810b78e9",
        "domain": "omni",
        "question": "Calculate the sum: $\\sum_{n=1}^{99} \\left(n^{3}+3n^{2}+3n\\right)$.",
        "answer": "25502400",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2417,
        "difficulty": 4.5
      },
      {
        "id": "5feb9689d3ff9240723648eed3b48dd8dcca313e8efcfb200cf7a12ca592c458",
        "domain": "omni",
        "question": "A ball inside a rectangular container of width 7 and height 12 is launched from the lower-left vertex of the container. It first strikes the right side of the container after traveling a distance of $\\sqrt{53}$ (and strikes no other sides between its launch and its impact with the right side). How many times does the ball bounce before it returns to a vertex? (The final contact with a vertex does not count as a bounce.)",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 1722,
        "difficulty": 5.25
      },
      {
        "id": "6f3156677d1a2c918e74de40035ce8eb42f876a1fcd5aa8fb56a9a6e4209c867",
        "domain": "omni",
        "question": "Find $\\prod_{n=2}^{\\infty}\\left(1-\\frac{1}{n^{2}}\\right)$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Calculus -> Techniques of Integration -> Other",
        "source": "HMMT_2",
        "source_index": 409,
        "difficulty": 3.5
      },
      {
        "id": "f899fd7262720a16f42568b208f5c15e25f55c651689ad73b80c968325e0b005",
        "domain": "omni",
        "question": "Let $P$ be a polyhedron where every face is a regular polygon, and every edge has length 1. Each vertex of $P$ is incident to two regular hexagons and one square. Choose a vertex $V$ of the polyhedron. Find the volume of the set of all points contained in $P$ that are closer to $V$ than to any other vertex.",
        "answer": "\\frac{\\sqrt{2}}{3}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 526,
        "difficulty": 4.0
      },
      {
        "id": "ac53fc1d113bbe459bf08ba7701d2a6d6ab71872739c87c16c04c1c39737e33c",
        "domain": "omni",
        "question": "There are $2022$ equally spaced points on a circular track $\\gamma$ of circumference $2022$. The points are labeled $A_1, A_2, \\ldots, A_{2022}$ in some order, each label used once. Initially, Bunbun the Bunny begins at $A_1$. She hops along $\\gamma$ from $A_1$ to $A_2$, then from $A_2$ to $A_3$, until she reaches $A_{2022}$, after which she hops back to $A_1$. When hopping from $P$ to $Q$, she always hops along the shorter of the two arcs $\\widehat{PQ}$ of $\\gamma$; if $\\overline{PQ}$ is a diameter of $\\gamma$, she moves along either semicircle.\n\nDetermine the maximal possible sum of the lengths of the $2022$ arcs which Bunbun traveled, over all possible labellings of the $2022$ points.\n\n[i]Kevin Cong[/i]",
        "answer": "2042222",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Other",
        "source": "usa_team_selection_test_for_imo",
        "source_index": 6,
        "difficulty": 8.0
      },
      {
        "id": "9fa417e1c067e06b9fe101543ab3f75b908ac64344d3124a056d46dab7db7421",
        "domain": "omni",
        "question": "A number is called [i]Norwegian[/i] if it has three distinct positive divisors whose sum is equal to $2022$. Determine the smallest Norwegian number.\n(Note: The total number of positive divisors of a Norwegian number is allowed to be larger than $3$.)",
        "answer": "1344",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Number Theory -> Other",
        "source": "imo_shortlist",
        "source_index": 3828,
        "difficulty": 6.5
      },
      {
        "id": "426837b039ac87f0efdce694bfdeaa2808d5110b52492123cb7cedc2cdaea1d5",
        "domain": "omni",
        "question": "A cuboctahedron is a polyhedron whose faces are squares and equilateral triangles such that two squares and two triangles alternate around each vertex. What is the volume of a cuboctahedron of side length 1?",
        "answer": "5 \\sqrt{2} / 3",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1005,
        "difficulty": 3.5
      },
      {
        "id": "03089325756f32aee644817f97496fd79296c41977584d0a1b8a0c217e54470e",
        "domain": "omni",
        "question": "Three distinct integers $a, b,$ and $c$ satisfy the following three conditions: $abc=17955$, $a, b,$ and $c$ form an arithmetic sequence in that order, and $(3a+b), (3b+c),$ and $(3c+a)$ form a geometric sequence in that order. What is the value of $a+b+c$?",
        "answer": "-63",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2722,
        "difficulty": 3.0
      },
      {
        "id": "b37d92a17a6db1fcc0c0f3c6c9857b2c690911f49b5e18ed94688a3426fd8afb",
        "domain": "omni",
        "question": "Find the number of arrangements of 4 beads (2 red, 2 green, 2 blue) in a circle such that the two red beads are not adjacent.",
        "answer": "11",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2419,
        "difficulty": 4.5
      },
      {
        "id": "0e9dc9a291a03569b799148257abe5976c94f862b9e19d654b2ed8ecfac5df78",
        "domain": "omni",
        "question": "Your math friend Steven rolls five fair icosahedral dice (each of which is labelled $1,2, \\ldots, 20$ on its sides). He conceals the results but tells you that at least half of the rolls are 20. Assuming that Steven is truthful, what is the probability that all three remaining concealed dice show $20 ?$",
        "answer": "\\frac{1}{58}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1113,
        "difficulty": 4.0
      },
      {
        "id": "69e079d3928c9b55d54c85cc461409084812930b8353c3b0a5d71b1f570733fd",
        "domain": "omni",
        "question": "In the octagon COMPUTER exhibited below, all interior angles are either $90^{\\circ}$ or $270^{\\circ}$ and we have $C O=O M=M P=P U=U T=T E=1$. Point $D$ (not to scale in the diagram) is selected on segment $R E$ so that polygons COMPUTED and $C D R$ have the same area. Find $D R$.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1815,
        "difficulty": 4.5
      },
      {
        "id": "f1675d831620a08db72ffbf5120d769ae4f1414c47f04719cf2555d53adfcd0e",
        "domain": "omni",
        "question": "Let $N=2^{(2^{2})}$ and $x$ be a real number such that $N^{(N^{N})}=2^{(2^{x})}$. Find $x$.",
        "answer": "66",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 1631,
        "difficulty": 5.5
      },
      {
        "id": "86d6fca35eb56a0c2128ad9cdf960d38e434f4707b66e10338e6901aebb607d4",
        "domain": "omni",
        "question": "How many of the four integers $222, 2222, 22222$, and $222222$ are multiples of 3?",
        "answer": "2",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "cayley",
        "source_index": 2782,
        "difficulty": 1.5
      },
      {
        "id": "250e492d06ef175c4849397706d1e65498042aaa9914e16853991975beb63a47",
        "domain": "omni",
        "question": "Let $m$ and $n$ be positive integers with $m\\le 2000$ and $k=3-\\frac{m}{n}$. Find the smallest positive value of $k$.",
        "answer": "$\\boxed{ \\frac{1}{667}} .$",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions; Mathematics -> Number Theory -> Prime Numbers",
        "source": "jbmo_shortlists",
        "source_index": 4113,
        "difficulty": 3.5
      },
      {
        "id": "206a768f2aa765c801583fb47358a80a968539753da2e588fc42c6d0234598df",
        "domain": "omni",
        "question": "Square \\(ABCD\\) is inscribed in circle \\(\\omega\\) with radius 10. Four additional squares are drawn inside \\(\\omega\\) but outside \\(ABCD\\) such that the lengths of their diagonals are as large as possible. A sixth square is drawn by connecting the centers of the four aforementioned small squares. Find the area of the sixth square.",
        "answer": "144",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 475,
        "difficulty": 5.0
      },
      {
        "id": "897fc5964ed9cf2dfd59979c75d2ae805e5bd1d4062fefbd138198325158b5bf",
        "domain": "omni",
        "question": "Suppose $a, b$, and $c$ are distinct positive integers such that $\\sqrt{a \\sqrt{b \\sqrt{c}}}$ is an integer. Compute the least possible value of $a+b+c$.",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1511,
        "difficulty": 4.5
      },
      {
        "id": "a7c1099c27f784b3b0561ed21242f946e45e5b7cb500efcf704cd70d58089077",
        "domain": "omni",
        "question": "What is the probability that Robbie will win if he and Francine each roll a special six-sided die three times, and after two rolls each, Robbie has a score of 8 and Francine has a score of 10?",
        "answer": "\\frac{55}{441}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 3075,
        "difficulty": 2.0
      },
      {
        "id": "3361d70d150924a739e0c617a36e50bcd9e8f4ca9b23d4ba3d8a9d4fafd26699",
        "domain": "omni",
        "question": "How many points $(x, y)$, with $x$ and $y$ both integers, are on the line with equation $y=4x+3$ and inside the region bounded by $x=25, x=75, y=120$, and $y=250$?",
        "answer": "32",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3089,
        "difficulty": 2.5
      },
      {
        "id": "67a084a2ee18312546e7470890de36c098a1b7a2a083894c78bd1c512173e066",
        "domain": "omni",
        "question": "Andrea flips a fair coin repeatedly, continuing until she either flips two heads in a row (the sequence $H H$ ) or flips tails followed by heads (the sequence $T H$ ). What is the probability that she will stop after flipping $H H$ ?",
        "answer": "1/4",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1119,
        "difficulty": 2.5
      },
      {
        "id": "9d76e9f9a5770ad89d95892951a4bc1bf684eb27c4373033c6d022ec17756262",
        "domain": "omni",
        "question": "Compute the sum of all integers $1 \\leq a \\leq 10$ with the following property: there exist integers $p$ and $q$ such that $p, q, p^{2}+a$ and $q^{2}+a$ are all distinct prime numbers.",
        "answer": "20",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1043,
        "difficulty": 3.5
      },
      {
        "id": "70d28a3379572a356dc8b939f941baf90e7fec60eba89d8b98d12abf1e969d15",
        "domain": "omni",
        "question": "Let $a_0 = 5/2$ and $a_k = a_{k-1}^2 - 2$ for $k \\geq 1$. Compute \\[ \\prod_{k=0}^\\infty \\left(1 - \\frac{1}{a_k} \\right) \\] in closed form.",
        "answer": "\\frac{3}{7}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "putnam",
        "source_index": 3245,
        "difficulty": 8.0
      },
      {
        "id": "986a51174d5f150ab691105acf84d298f9c99062eb6fb4c6cc78ec15cd2eb0eb",
        "domain": "omni",
        "question": "Which of the following divisions is not equal to a whole number: $\\frac{60}{12}$, $\\frac{60}{8}$, $\\frac{60}{5}$, $\\frac{60}{4}$, $\\frac{60}{3}$?",
        "answer": "7.5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3444,
        "difficulty": 1.0
      },
      {
        "id": "92f5822603fb9da4b5035b98ba2641a12c05e4031116fbbaff9fec257de9911d",
        "domain": "omni",
        "question": "Let $n$ be a positive integer. Let there be $P_{n}$ ways for Pretty Penny to make exactly $n$ dollars out of quarters, dimes, nickels, and pennies. Also, let there be $B_{n}$ ways for Beautiful Bill to make exactly $n$ dollars out of one dollar bills, quarters, dimes, and nickels. As $n$ goes to infinity, the sequence of fractions \\frac{P_{n}}{B_{n}}$ approaches a real number $c$. Find $c$.",
        "answer": "20",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other; Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1933,
        "difficulty": 4.5
      },
      {
        "id": "73a90d870cd41a7c3642af50f9de077fce919e14bb0bb821bfd798d070538728",
        "domain": "omni",
        "question": "The product \\( \\left(1-\\frac{1}{3}\\right)\\left(1-\\frac{1}{4}\\right)\\left(1-\\frac{1}{5}\\right) \\) is equal to what?",
        "answer": "\\frac{2}{5}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3392,
        "difficulty": 1.5
      },
      {
        "id": "b498b2c4c05faa84a7458d06e37835e2d78a78db28c11ec403cbe94d56c91579",
        "domain": "omni",
        "question": "Charlie folds an $\\frac{17}{2}$-inch by 11-inch piece of paper in half twice, each time along a straight line parallel to one of the paper's edges. What is the smallest possible perimeter of the piece after two such folds?",
        "answer": "$\\frac{39}{2}$",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2217,
        "difficulty": 3.5
      },
      {
        "id": "af7594af1f402112fc44235c83d67e5317e70a243df8a0e5df2d151537000ff9",
        "domain": "omni",
        "question": "12 points are placed around the circumference of a circle. How many ways are there to draw 6 non-intersecting chords joining these points in pairs?",
        "answer": "132",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_2",
        "source_index": 454,
        "difficulty": 5.0
      },
      {
        "id": "0f4e11a1bc321d26f910c174bcb70f956c16632404201e02a14826e0fdbd92a1",
        "domain": "omni",
        "question": "James is standing at the point $(0,1)$ on the coordinate plane and wants to eat a hamburger. For each integer $n \\geq 0$, the point $(n, 0)$ has a hamburger with $n$ patties. There is also a wall at $y=2.1$ which James cannot cross. In each move, James can go either up, right, or down 1 unit as long as he does not cross the wall or visit a point he has already visited. Every second, James chooses a valid move uniformly at random, until he reaches a point with a hamburger. Then he eats the hamburger and stops moving. Find the expected number of patties that James eats on his burger.",
        "answer": "\\frac{7}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1996,
        "difficulty": 4.5
      },
      {
        "id": "4783648d8f827f30fd7f8807b82bd490201cc33cc7e1c2318a621f7aab8e2457",
        "domain": "omni",
        "question": "A permutation of the set of positive integers $[n] = \\{1, 2, \\ldots, n\\}$ is a sequence $(a_1, a_2, \\ldots, a_n)$ such that each element of $[n]$ appears precisely one time as a term of the sequence. For example, $(3, 5, 1, 2, 4)$ is a permutation of $[5]$ . Let $P(n)$ be the number of permutations of $[n]$ for which $ka_k$ is a perfect square for all $1\\leq k\\leq n$ . Find with proof the smallest $n$ such that $P(n)$ is a multiple of $2010$ .",
        "answer": "\\(\\boxed{4489}\\)",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations; Mathematics -> Number Theory -> Perfect Squares -> Other",
        "source": "usajmo",
        "source_index": 291,
        "difficulty": 6.5
      },
      {
        "id": "b3a574f50e161da252317d489c74ff2ed7de34c318adc4e569cf21ef1599e722",
        "domain": "omni",
        "question": "Suppose that $(a_1,b_1),$ $(a_2,b_2),$ $\\dots,$ $(a_{100},b_{100})$ are distinct ordered pairs of nonnegative integers. Let $N$ denote the number of pairs of integers $(i,j)$ satisfying $1\\leq i<j\\leq 100$ and $|a_ib_j-a_jb_i|=1$. Determine the largest possible value of $N$ over all possible choices of the $100$ ordered pairs.",
        "answer": "197",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usomo",
        "source_index": 3575,
        "difficulty": 8.0
      },
      {
        "id": "8f4dbf6574d89e54dd0be321f46003c060a452e004b2f00a007c90e8968da209",
        "domain": "omni",
        "question": "Let $A$ be a set of integers such that for each integer $m$, there exists an integer $a \\in A$ and positive integer $n$ such that $a^{n} \\equiv m(\\bmod 100)$. What is the smallest possible value of $|A|$?",
        "answer": "41",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory",
        "source": "HMMT_2",
        "source_index": 668,
        "difficulty": 5.0
      },
      {
        "id": "8af3e9a0cba2820d28437090f0cb5e2d46c9f3f5cb55c976c766e34ace8a8949",
        "domain": "omni",
        "question": "Pyramid $E A R L Y$ is placed in $(x, y, z)$ coordinates so that $E=(10,10,0), A=(10,-10,0), R=(-10,-10,0), L=(-10,10,0)$, and $Y=(0,0,10)$. Tunnels are drilled through the pyramid in such a way that one can move from $(x, y, z)$ to any of the 9 points $(x, y, z-1),(x \\pm 1, y, z-1)$, $(x, y \\pm 1, z-1),(x \\pm 1, y \\pm 1, z-1)$. Sean starts at $Y$ and moves randomly down to the base of the pyramid, choosing each of the possible paths with probability $\\frac{1}{9}$ each time. What is the probability that he ends up at the point $(8,9,0)$?",
        "answer": "\\frac{550}{9^{10}}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1456,
        "difficulty": 4.0
      },
      {
        "id": "736e3ac385115c1d7ad517aebb4b0571dd4218f2fb43031153a0b0b92b1733d9",
        "domain": "omni",
        "question": "An empty $2020 \\times 2020 \\times 2020$ cube is given, and a $2020 \\times 2020$ grid of square unit cells is drawn  on each of its six faces. A [i]beam[/i] is a $1 \\times 1 \\times 2020$ rectangular prism. Several beams are placed inside the cube subject to the following conditions:\n[list=]\n[*]The two $1 \\times 1$ faces of each beam coincide with unit cells lying on opposite faces of the cube. (Hence, there are $3 \\cdot {2020}^2$ possible positions for a beam.)\n[*]No two beams have intersecting interiors.\n[*]The interiors of each of the four $1 \\times 2020$ faces of each beam touch either a face of the cube or the interior of the face of another beam.\n[/list]\nWhat is the smallest positive number of beams that can be placed to satisfy these conditions?",
        "answer": "3030",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usomo",
        "source_index": 3597,
        "difficulty": 8.0
      },
      {
        "id": "5e66533e41926e505dde7d8db2c360538e9c48f9de366331b1bcfcbbea407919",
        "domain": "omni",
        "question": "Calculate the value of the expression $(8 \\times 6)-(4 \\div 2)$.",
        "answer": "46",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2837,
        "difficulty": 1.0
      },
      {
        "id": "ce39c9df58b989d4f9ebdb3e716f2c605d66eec9dd5c06c1bb1eb7fd19948c2d",
        "domain": "omni",
        "question": "For how many of the given drawings can the six dots be labelled to represent the links between suspects?",
        "answer": "2",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "pascal",
        "source_index": 2956,
        "difficulty": 2.0
      },
      {
        "id": "3b139cb5b2bb57920d86dccf5738456d3c6434cd4d7cb25fa5925ae592dd21bb",
        "domain": "omni",
        "question": "For breakfast, Milan is eating a piece of toast shaped like an equilateral triangle. On the piece of toast rests a single sesame seed that is one inch away from one side, two inches away from another side, and four inches away from the third side. He places a circular piece of cheese on top of the toast that is tangent to each side of the triangle. What is the area of this piece of cheese?",
        "answer": "\\frac{49 \\pi}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Areas -> Other",
        "source": "HMMT_11",
        "source_index": 2154,
        "difficulty": 4.0
      },
      {
        "id": "4dc88055f2ae7b3027a8dc0db4e2d0b233af3b4ea26bac8a13005cab37363f48",
        "domain": "omni",
        "question": "How many perfect squares divide $2^{3} \\cdot 3^{5} \\cdot 5^{7} \\cdot 7^{9}$?",
        "answer": "120",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1550,
        "difficulty": 5.0
      },
      {
        "id": "3072b9360c56b189dffba11f706e4fe0f48f740d65fda1f51f0fe0eda8fa326d",
        "domain": "omni",
        "question": "Consider triangle $A B C$ with side lengths $A B=4, B C=7$, and $A C=8$. Let $M$ be the midpoint of segment $A B$, and let $N$ be the point on the interior of segment $A C$ that also lies on the circumcircle of triangle $M B C$. Compute $B N$.",
        "answer": "\\frac{\\sqrt{210}}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 1941,
        "difficulty": 5.0
      },
      {
        "id": "d7d9aff381b12fbe5d449af450861ff62b3106800719cd0ef6ef8e187907a9d5",
        "domain": "omni",
        "question": "Compute the number of even positive integers $n \\leq 2024$ such that $1,2, \\ldots, n$ can be split into $\\frac{n}{2}$ pairs, and the sum of the numbers in each pair is a multiple of 3.",
        "answer": "675",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 700,
        "difficulty": 5.5
      },
      {
        "id": "0c9c52aeaa0e348a91ab1c20bf975ea964ccef56216bb99c3a029cc1712c990c",
        "domain": "omni",
        "question": "On a chessboard, a queen attacks every square it can reach by moving from its current square along a row, column, or diagonal without passing through a different square that is occupied by a chess piece. Find the number of ways in which three indistinguishable queens can be placed on an $8 \\times 8$ chess board so that each queen attacks both others.",
        "answer": "864",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2553,
        "difficulty": 5.0
      },
      {
        "id": "ab2ad361184107a806b4049d7f3a46fcd52ca0ca0de03377d3ad897aac089a98",
        "domain": "omni",
        "question": "The equation\n$$(x-1)(x-2)\\cdots(x-2016)=(x-1)(x-2)\\cdots (x-2016)$$\nis written on the board, with $2016$ linear factors on each side. What is the least possible value of $k$ for which it is possible to erase exactly $k$ of these $4032$ linear factors so that at least one factor remains on each side and the resulting equation has no real solutions?",
        "answer": "2016",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "imo",
        "source_index": 3851,
        "difficulty": 6.0
      },
      {
        "id": "278531ca6e78451449aa3ab1afb3665cb1c0bfb3b6e0a01100f1c46bd192ebb4",
        "domain": "omni",
        "question": "Let $A B C D$ be a rectangle with $A B=6$ and $B C=4$. Let $E$ be the point on $B C$ with $B E=3$, and let $F$ be the point on segment $A E$ such that $F$ lies halfway between the segments $A B$ and $C D$. If $G$ is the point of intersection of $D F$ and $B C$, find $B G$.",
        "answer": "1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1936,
        "difficulty": 4.0
      },
      {
        "id": "cb10e227cd7b94f436eaee71a7e44e86ad2e38fd84813190e7f0197bd2632886",
        "domain": "omni",
        "question": "Tessa has a unit cube, on which each vertex is labeled by a distinct integer between 1 and 8 inclusive. She also has a deck of 8 cards, 4 of which are black and 4 of which are white. At each step she draws a card from the deck, and if the card is black, she simultaneously replaces the number on each vertex by the sum of the three numbers on vertices that are distance 1 away from this vertex; if the card is white, she simultaneously replaces the number on each vertex by the sum of the three numbers on vertices that are distance \\sqrt{2} away from this vertex. When Tessa finishes drawing all cards of the deck, what is the maximum possible value of a number that is on the cube?",
        "answer": "42648",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms; Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_11",
        "source_index": 2264,
        "difficulty": 5.0
      },
      {
        "id": "11aeef773ab833dc63c2072a36afade5b745b3c5726351707441cf318a41474d",
        "domain": "omni",
        "question": "How many nonempty subsets of $\\{1,2,3, \\ldots, 12\\}$ have the property that the sum of the largest element and the smallest element is 13?",
        "answer": "1365",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1046,
        "difficulty": 4.0
      },
      {
        "id": "9b7bdd8fa71a28f75f96c2bd7433f830d505b1d2e3c1839523b802ce20fe438c",
        "domain": "omni",
        "question": "How many positive integers less than 100 are relatively prime to 200?",
        "answer": "40",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 2290,
        "difficulty": 4.0
      },
      {
        "id": "a951911f881c81b4fbd0ffed8da6af1f2428cf846bc5b0fc7a498133df5f6fc8",
        "domain": "omni",
        "question": "A pizza is cut into 10 pieces. Two of the pieces are each \\(\\frac{1}{24}\\) of the whole pizza, four are each \\(\\frac{1}{12}\\), two are each \\(\\frac{1}{8}\\), and two are each \\(\\frac{1}{6}\\). A group of \\(n\\) friends share the pizza by distributing all of these pieces. They do not cut any of these pieces. Each of the \\(n\\) friends receives, in total, an equal fraction of the whole pizza. What is the sum of the values of \\(n\\) with \\(2 \\leq n \\leq 10\\) for which this is not possible?",
        "answer": "39",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3185,
        "difficulty": 2.0
      },
      {
        "id": "62d1dbbd835be20edf7b73b72bf0705c6025a27dcb9cd3f06c958d7941669cb3",
        "domain": "omni",
        "question": "If two fair dice are tossed, what is the probability that their sum is divisible by 5 ?",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 377,
        "difficulty": 4.0
      },
      {
        "id": "eb5a1d5b05f58dd8b8e758419c7c6b99064c87130da20a9949a4e6ac59d27c38",
        "domain": "omni",
        "question": "$ S$ is a non-empty subset of the set $ \\{ 1, 2, \\cdots, 108 \\}$, satisfying:\r\n\r\n(1) For any two numbers $ a,b \\in S$ ( may not distinct), there exists $ c \\in S$, such that $ \\gcd(a,c)\\equal{}\\gcd(b,c)\\equal{}1$.\r\n\r\n(2) For any two numbers $ a,b \\in S$ ( may not distinct), there exists $ c' \\in S$, $ c' \\neq a$, $ c' \\neq b$, such that $ \\gcd(a, c') > 1$, $ \\gcd(b,c') >1$.\r\n\r\nFind the largest possible value of $ |S|$.",
        "answer": "79",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 85,
        "difficulty": 8.0
      },
      {
        "id": "d09021bca728e123820f34a2ea5de8fd05e7cb3f57ea0ccfe85c1c46a40c2c66",
        "domain": "omni",
        "question": "Let $a, b, c$ be nonzero real numbers such that $a+b+c=0$ and $a^{3}+b^{3}+c^{3}=a^{5}+b^{5}+c^{5}$. Find the value of $a^{2}+b^{2}+c^{2}$.",
        "answer": "\\frac{6}{5}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 518,
        "difficulty": 4.0
      },
      {
        "id": "a98f3bc5c230c71f081b01cc870ac056dd3de4c23947c9ddfc3631b7d3ae7cbe",
        "domain": "omni",
        "question": "Define a sequence $a_{i, j}$ of integers such that $a_{1, n}=n^{n}$ for $n \\geq 1$ and $a_{i, j}=a_{i-1, j}+a_{i-1, j+1}$ for all $i, j \\geq 1$. Find the last (decimal) digit of $a_{128,1}$.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 631,
        "difficulty": 5.0
      },
      {
        "id": "2237f104d09b1a79c6bb4aaee196f69e580b420a56a5e5d35d62a12c9297d124",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=23, B C=24$, and $C A=27$. Let $D$ be the point on segment $A C$ such that the incircles of triangles $B A D$ and $B C D$ are tangent. Determine the ratio $C D / D A$.",
        "answer": "$\\frac{14}{13}$",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1979,
        "difficulty": 4.5
      },
      {
        "id": "2b99bc4af59d517ab11d38cdebbadc254dfbc0266ab8ee203e10efa1bfe60423",
        "domain": "omni",
        "question": "In a number line, point $P$ is at 3 and $V$ is at 33. The number line between 3 and 33 is divided into six equal parts by the points $Q, R, S, T, U$. What is the sum of the lengths of $PS$ and $TV$?",
        "answer": "25",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "pascal",
        "source_index": 2923,
        "difficulty": 2.0
      },
      {
        "id": "f853976444d0f5ccf51e03ac2531811658100a7efb30326ca0d535a3c5af34db",
        "domain": "omni",
        "question": "Compute the sum of all two-digit positive integers $x$ such that for all three-digit (base 10) positive integers \\underline{a} \\underline{b} \\underline{c}, if \\underline{a} \\underline{b} \\underline{c} is a multiple of $x$, then the three-digit (base 10) number \\underline{b} \\underline{c} \\underline{a} is also a multiple of $x$.",
        "answer": "64",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1661,
        "difficulty": 4.5
      },
      {
        "id": "2c19e93f073e94f580440f82e637134a0dbb674f64d46fa34f288cbccc84f4d1",
        "domain": "omni",
        "question": "Suppose \\(\\triangle A B C\\) has lengths \\(A B=5, B C=8\\), and \\(C A=7\\), and let \\(\\omega\\) be the circumcircle of \\(\\triangle A B C\\). Let \\(X\\) be the second intersection of the external angle bisector of \\(\\angle B\\) with \\(\\omega\\), and let \\(Y\\) be the foot of the perpendicular from \\(X\\) to \\(B C\\). Find the length of \\(Y C\\).",
        "answer": "\\frac{13}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1553,
        "difficulty": 5.0
      },
      {
        "id": "75d796ead63b4c9afaf7bc703b57d29f2908ee172de0f6eefeaf8de0165bac40",
        "domain": "omni",
        "question": "Let $ S \\equal{} \\{1,2,3,\\cdots ,280\\}$. Find the smallest integer $ n$ such that each $ n$-element subset of $ S$ contains five numbers which are pairwise relatively prime.",
        "answer": "217",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "imo",
        "source_index": 3844,
        "difficulty": 6.5
      },
      {
        "id": "70e3018d2b2df8f7c823d48da2d22a91eeb5b38c53c3561372c07a76e9547f83",
        "domain": "omni",
        "question": "Let $A B C$ be an equilateral triangle with $A B=3$. Circle $\\omega$ with diameter 1 is drawn inside the triangle such that it is tangent to sides $A B$ and $A C$. Let $P$ be a point on $\\omega$ and $Q$ be a point on segment $B C$. Find the minimum possible length of the segment $P Q$.",
        "answer": "\\frac{3 \\sqrt{3}-3}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 2086,
        "difficulty": 4.0
      },
      {
        "id": "8ae58e723c9e0ec7ad6492a2c3b59c9ce0c69bea8f4e70ca5af9832ba08bd840",
        "domain": "omni",
        "question": "Cyclic pentagon $ABCDE$ has side lengths $AB=BC=5, CD=DE=12$, and $AE=14$. Determine the radius of its circumcircle.",
        "answer": "\\frac{225\\sqrt{11}}{88}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 501,
        "difficulty": 5.0
      },
      {
        "id": "5908c5a0af2dd9a20d25673c0273046e0e0f9e020a3b8327070492a69b8e0ebc",
        "domain": "omni",
        "question": "A triangle with side lengths $5,7,8$ is inscribed in a circle $C$. The diameters of $C$ parallel to the sides of lengths 5 and 8 divide $C$ into four sectors. What is the area of either of the two smaller ones?",
        "answer": "\\frac{49}{18} \\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2508,
        "difficulty": 4.5
      },
      {
        "id": "0a0e966d21e072b9f1eb6e01fdf76c3217b98fe6dc77c372be2b39dfb34db293",
        "domain": "omni",
        "question": "The sum of the digits of the time 19 minutes ago is two less than the sum of the digits of the time right now. Find the sum of the digits of the time in 19 minutes. (Here, we use a standard 12-hour clock of the form hh:mm.)",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2523,
        "difficulty": 3.5
      },
      {
        "id": "48acd7dc87a2c193f5580bcb92950c419c65455406cbe786e260016585ba1e3f",
        "domain": "omni",
        "question": "Compute the product of all positive integers $b \\geq 2$ for which the base $b$ number $111111_{b}$ has exactly $b$ distinct prime divisors.",
        "answer": "24",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 951,
        "difficulty": 4.5
      },
      {
        "id": "bfb3600d936d73809ebd577a8bad5be6c9c40560086bce4b579e2322b1d7a8ce",
        "domain": "omni",
        "question": "Find the shortest distance between the lines $\\frac{x+2}{2}=\\frac{y-1}{3}=\\frac{z}{1}$ and $\\frac{x-3}{-1}=\\frac{y}{1}=\\frac{z+1}{2}$",
        "answer": "\\frac{5 \\sqrt{3}}{3}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 1752,
        "difficulty": 3.5
      },
      {
        "id": "38625eb4c55a5ad55ddbf8c6d88ec1299920ff407305cdd2173b39e2cf5d92e3",
        "domain": "omni",
        "question": "Last summer, Pat worked at a summer camp. For each day that he worked, he earned \\$100 and he was not charged for food. For each day that he did not work, he was not paid and he was charged \\$20 for food. After 70 days, the money that he earned minus his food costs equalled \\$5440. On how many of these 70 days did Pat work?",
        "answer": "57",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3091,
        "difficulty": 1.5
      },
      {
        "id": "3d3aa2f7a86917ec6ae4c3fb4d104d239fe5c78695c047a617cdba96d95c1efe",
        "domain": "omni",
        "question": "Let $\\pi$ be a permutation of the numbers from 1 through 2012. What is the maximum possible number of integers $n$ with $1 \\leq n \\leq 2011$ such that $\\pi(n)$ divides $\\pi(n+1)$?",
        "answer": "1006",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2006,
        "difficulty": 5.0
      },
      {
        "id": "1fedce80b66396477897338a2dec177d4b34d274c62da6a50289e7a2732fc602",
        "domain": "omni",
        "question": "The numbers $5,6,10,17$, and 21 are rearranged so that the sum of the first three numbers is equal to the sum of the last three numbers. Which number is in the middle of this rearrangement?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3424,
        "difficulty": 2.0
      },
      {
        "id": "d78cf852928d09ad27ac3873f899eb1b455abdb2c79b9a261db4e11444513b19",
        "domain": "omni",
        "question": "All positive integers whose binary representations (excluding leading zeroes) have at least as many 1's as 0's are put in increasing order. Compute the number of digits in the binary representation of the 200th number.",
        "answer": "9",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2295,
        "difficulty": 5.0
      },
      {
        "id": "151254190352b43f545f96b7cd6a33e7d4c44a3ca666946e55ab90c1ccc63ce9",
        "domain": "omni",
        "question": "Let $b(x)=x^{2}+x+1$. The polynomial $x^{2015}+x^{2014}+\\cdots+x+1$ has a unique \"base $b(x)$ \" representation $x^{2015}+x^{2014}+\\cdots+x+1=\\sum_{k=0}^{N} a_{k}(x) b(x)^{k}$ where each \"digit\" $a_{k}(x)$ is either the zero polynomial or a nonzero polynomial of degree less than $\\operatorname{deg} b=2$; and the \"leading digit $a_{N}(x)$ \" is nonzero. Find $a_{N}(0)$.",
        "answer": "-1006",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 737,
        "difficulty": 4.0
      },
      {
        "id": "0ffedd32ea90c80fce35a92ae4dae2d6be0861b0c449a829786327686317b800",
        "domain": "omni",
        "question": "The three sides of a right triangle form a geometric sequence. Determine the ratio of the length of the hypotenuse to the length of the shorter leg.",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions; Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_11",
        "source_index": 1809,
        "difficulty": 4.0
      },
      {
        "id": "c728eb46ab70e673fd578a17b55059b52ff8c44ca327ffff4b914d244da7fef8",
        "domain": "omni",
        "question": "Two fair six-sided dice are rolled. What is the probability that their sum is at least 10?",
        "answer": "\\frac{1}{6}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2286,
        "difficulty": 3.5
      },
      {
        "id": "5b47ce3aa3a7c1eded3d17fc5ff257d893309bc2df61f30cef0309e4f5d43bc9",
        "domain": "omni",
        "question": "Find the largest real number $k$ such that there exists a sequence of positive reals $\\left\\{a_{i}\\right\\}$ for which $\\sum_{n=1}^{\\infty} a_{n}$ converges but $\\sum_{n=1}^{\\infty} \\frac{\\sqrt{a_{n}}}{n^{k}}$ does not.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1962,
        "difficulty": 4.0
      },
      {
        "id": "f3e9f1cf1a19f0d77ca112af88e476eb173c6cb4f25328b9c9b8d10e0016a11f",
        "domain": "omni",
        "question": "In a game show, Bob is faced with 7 doors, 2 of which hide prizes. After he chooses a door, the host opens three other doors, of which one is hiding a prize. Bob chooses to switch to another door. What is the probability that his new door is hiding a prize?",
        "answer": "\\frac{5}{21}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 531,
        "difficulty": 5.0
      },
      {
        "id": "f78afa49a79155c692535f90c10d0355797c860b75e108c8039a57492f4c7c02",
        "domain": "omni",
        "question": "Find the sum of $\\frac{1}{n}$ over all positive integers $n$ with the property that the decimal representation of $\\frac{1}{n}$ terminates.",
        "answer": "\\sqrt{\\frac{5}{2}}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2288,
        "difficulty": 3.5
      },
      {
        "id": "4ae254c36b99191edb251640f8425fcfb41e62b8a561868fdab161b7bb92e242",
        "domain": "omni",
        "question": "If $x$, $y$, $z$ are positive numbers satisfying \n\\[x+\\frac{y}{z}=y+\\frac{z}{x}=z+\\frac{x}{y}=2.\\]\nFind all the possible values of $x+y+z$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "centroamerican",
        "source_index": 3562,
        "difficulty": 5.0
      },
      {
        "id": "ca6df9eb49ecfed7bed139faa78ca8c07248eb8a40b358ea1333521f6d65fba3",
        "domain": "omni",
        "question": "Find $a_{2012}$ if $a_{n} \\equiv a_{n-1}+n(\\bmod 2012)$ and $a_{1}=1$.",
        "answer": "1006",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2356,
        "difficulty": 3.5
      },
      {
        "id": "ca71e94da09a256758d1d4c49e2924a3945d93fc3cb51cbada5b93e623be8bd4",
        "domain": "omni",
        "question": "If each of Bill's steps is $\frac{1}{2}$ metre long, how many steps does Bill take to walk 12 metres in a straight line?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals",
        "source": "cayley",
        "source_index": 3152,
        "difficulty": 1.0
      },
      {
        "id": "fea88ab94666ac6052dd53eb70507cc0a135451fe4babfada853f51b7307d36d",
        "domain": "omni",
        "question": "Let $a_{0}, a_{1}, a_{2}, \\ldots$ be an infinite sequence where each term is independently and uniformly random in the set $\\{1,2,3,4\\}$. Define an infinite sequence $b_{0}, b_{1}, b_{2}, \\ldots$ recursively by $b_{0}=1$ and $b_{i+1}=a_{i}^{b_{i}}$. Compute the expected value of the smallest positive integer $k$ such that $b_{k} \\equiv 1(\\bmod 5)$.",
        "answer": "\\frac{35}{16}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1574,
        "difficulty": 4.5
      },
      {
        "id": "ed11752f3856f36648b4aa54008ecb78b15e55d9c81bb8c5440980e831631048",
        "domain": "omni",
        "question": "Compute the remainder when 10002000400080016003200640128025605121024204840968192 is divided by 100020004000800160032.",
        "answer": "40968192",
        "topic": "Mathematics -> Algebra -> Numbers -> Other; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1307,
        "difficulty": 5.0
      },
      {
        "id": "09e7116a0b2c77988b3b23b2aeb01e093a39e179826b47eec971185fd45871ae",
        "domain": "omni",
        "question": "Compute the largest positive integer such that $\\frac{2007!}{2007^{n}}$ is an integer.",
        "answer": "9",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1672,
        "difficulty": 5.0
      },
      {
        "id": "d22667788f7213586750a7cdde96b3cc6b8c253bc2fc6987b24c19437d484d8b",
        "domain": "omni",
        "question": "Another professor enters the same room and says, 'Each of you has to write down an integer between 0 and 200. I will then compute $X$, the number that is 3 greater than half the average of all the numbers that you will have written down. Each student who writes down the number closest to $X$ (either above or below $X$) will receive a prize.' One student, who misunderstood the question, announces to the class that he will write the number 107. If among the other 99 students it is common knowledge that all 99 of them will write down the best response, and there is no further communication between students, what single integer should each of the 99 students write down?",
        "answer": "7",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 759,
        "difficulty": 4.0
      },
      {
        "id": "8ab4daac5e11bdf2f7ca9b9a9459347a4027869c3257b5267bd3563cb65f1e0a",
        "domain": "omni",
        "question": "Alice and Bob are playing in the forest. They have six sticks of length $1,2,3,4,5,6$ inches. Somehow, they have managed to arrange these sticks, such that they form the sides of an equiangular hexagon. Compute the sum of all possible values of the area of this hexagon.",
        "answer": "33 \\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2262,
        "difficulty": 4.0
      },
      {
        "id": "0f6a8678c16d42ca916d527adc9185263743c9ce8676350af3e485208560987e",
        "domain": "omni",
        "question": "Let $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ be a function satisfying the following conditions: (a) $f(1)=1$ (b) $f(a) \\leq f(b)$ whenever $a$ and $b$ are positive integers with $a \\leq b$. (c) $f(2a)=f(a)+1$ for all positive integers $a$. How many possible values can the 2014-tuple $(f(1), f(2), \\ldots, f(2014))$ take?",
        "answer": "1007",
        "topic": "Mathematics -> Algebra -> Functions and Sequences -> Other",
        "source": "HMMT_2",
        "source_index": 979,
        "difficulty": 5.25
      },
      {
        "id": "f2392c26b6e30aeb3f276761a2bfbfd31b033b6ccaf79cf881227ccae2dcedc1",
        "domain": "omni",
        "question": "If $a$ and $b$ are positive integers that can each be written as a sum of two squares, then $a b$ is also a sum of two squares. Find the smallest positive integer $c$ such that $c=a b$, where $a=x^{3}+y^{3}$ and $b=x^{3}+y^{3}$ each have solutions in integers $(x, y)$, but $c=x^{3}+y^{3}$ does not.",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 396,
        "difficulty": 4.5
      },
      {
        "id": "02d58f7caaa26047a915dea031e493d5b974882adf3e14800f0237aa54ab7906",
        "domain": "omni",
        "question": "Determine the number of triples $0 \\leq k, m, n \\leq 100$ of integers such that $$ 2^{m} n-2^{n} m=2^{k} $$",
        "answer": "22",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1101,
        "difficulty": 5.5
      },
      {
        "id": "297805709aa50c7ac9225269b695ded098c2acc8532376b2efab36b1087339d9",
        "domain": "omni",
        "question": "Joe has written 5 questions of different difficulties for a test with problems numbered 1 though 5. He wants to make sure that problem $i$ is harder than problem $j$ whenever $i-j \\geq 3$. In how many ways can he order the problems for his test?",
        "answer": "25",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1944,
        "difficulty": 5.0
      },
      {
        "id": "f525a3f0f740b26dde2b4f030aae68ca237417652b6f9ee676ef23ecaa976295",
        "domain": "omni",
        "question": "Ainsley and Buddy play a game where they repeatedly roll a standard fair six-sided die. Ainsley wins if two multiples of 3 in a row are rolled before a non-multiple of 3 followed by a multiple of 3, and Buddy wins otherwise. If the probability that Ainsley wins is $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100a+b$.",
        "answer": "109",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2656,
        "difficulty": 3.5
      },
      {
        "id": "08c53e64f9769b79bbb5afe81d8a75a99b8b9bd28195ed0b21157136e0dfa286",
        "domain": "omni",
        "question": "Circle $\\Omega$ has radius 13. Circle $\\omega$ has radius 14 and its center $P$ lies on the boundary of circle $\\Omega$. Points $A$ and $B$ lie on $\\Omega$ such that chord $A B$ has length 24 and is tangent to $\\omega$ at point $T$. Find $A T \\cdot B T$.",
        "answer": "56",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 1521,
        "difficulty": 5.25
      },
      {
        "id": "f206d61a286f7835ab21090d5c07dac4a83926a1fc8d13f36fff0950b42884ac",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that $1^{2}+2^{2}+3^{2}+4^{2}+\\cdots+n^{2}$ is divisible by 100.",
        "answer": "24",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 446,
        "difficulty": 3.5
      },
      {
        "id": "3aa6edc4811faa686d46a4da96ceb6c6eff2c85f72679d83bf41e3852826759f",
        "domain": "omni",
        "question": "A particular coin has a $\\frac{1}{3}$ chance of landing on heads (H), $\\frac{1}{3}$ chance of landing on tails (T), and $\\frac{1}{3}$ chance of landing vertically in the middle (M). When continuously flipping this coin, what is the probability of observing the continuous sequence HMMT before HMT?",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1740,
        "difficulty": 5.0
      },
      {
        "id": "b3d7b09d9efcdf3c510d544e6455ce9c90db7373291243af46e4726ff66f6220",
        "domain": "omni",
        "question": "What is the value of \\( z \\) in the carpet installation cost chart?",
        "answer": "1261.40",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2884,
        "difficulty": 1.0
      },
      {
        "id": "54c775950d8a68f2912690a8d40ffd5f4c5b3fbb805e5af7e5eda9410aeb942e",
        "domain": "omni",
        "question": "Solve $x=\\sqrt{x-\\frac{1}{x}}+\\sqrt{1-\\frac{1}{x}}$ for $x$.",
        "answer": "\\frac{1+\\sqrt{5}}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 3310,
        "difficulty": 5.0
      },
      {
        "id": "c4dfa1ec662ce192b4fd779c57bdeb0fbcbf3abe0b88352c034e9ffe2e4a23ee",
        "domain": "omni",
        "question": "Suppose that $ABC$ is an isosceles triangle with $AB=AC$. Let $P$ be the point on side $AC$ so that $AP=2CP$. Given that $BP=1$, determine the maximum possible area of $ABC$.",
        "answer": "\\frac{9}{10}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 643,
        "difficulty": 5.25
      },
      {
        "id": "fe652cec3f41db9ac52773f88838841b1df5a9fbf56dbfe96af2068de1f19488",
        "domain": "omni",
        "question": "Find the greatest constant $\\lambda$ such that for any doubly stochastic matrix of order 100, we can pick $150$ entries such that if the other $9850$ entries were replaced by $0$, the sum of entries in each row and each column is at least $\\lambda$.\n\nNote: A doubly stochastic matrix of order $n$ is a $n\\times n$ matrix, all entries are nonnegative reals, and the sum of entries in each row and column is equal to 1.",
        "answer": "\\frac{17}{1900}",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Matrices; Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Algebra -> Other (Matrix-related optimization) -> Other",
        "source": "china_team_selection_test",
        "source_index": 45,
        "difficulty": 8.0
      },
      {
        "id": "2dca75ab7c6f1f43a0f277160cf495b6e9bdc2c88a8bde46a0a74c8afcc3127c",
        "domain": "omni",
        "question": "Find the smallest positive number $\\lambda $ , such that for any complex numbers ${z_1},{z_2},{z_3}\\in\\{z\\in C\\big| |z|<1\\}$ ,if  $z_1+z_2+z_3=0$, then $$\\left|z_1z_2 +z_2z_3+z_3z_1\\right|^2+\\left|z_1z_2z_3\\right|^2 <\\lambda .$$",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "china_team_selection_test",
        "source_index": 119,
        "difficulty": 9.0
      },
      {
        "id": "7f7e8010b6fddf6e35f02690d7fdfa8f82a4d79790e9b1badf4a2837132933b5",
        "domain": "omni",
        "question": "Victoria wants to order at least 550 donuts from Dunkin' Donuts for the HMMT 2014 November contest. However, donuts only come in multiples of twelve. Assuming every twelve donuts cost \\$7.49, what is the minimum amount Victoria needs to pay, in dollars?",
        "answer": "344.54",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1693,
        "difficulty": 4.0
      },
      {
        "id": "8a9ecb23169f6ad36454e947ea1069e69647905b8fc86cfd2c9f9ac7c1251fdd",
        "domain": "omni",
        "question": "Find the area of triangle $EFC$ given that $[EFC]=\\left(\\frac{5}{6}\\right)[AEC]=\\left(\\frac{5}{6}\\right)\\left(\\frac{4}{5}\\right)[ADC]=\\left(\\frac{5}{6}\\right)\\left(\\frac{4}{5}\\right)\\left(\\frac{2}{3}\\right)[ABC]$ and $[ABC]=20\\sqrt{3}$.",
        "answer": "\\frac{80\\sqrt{3}}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2378,
        "difficulty": 3.5
      },
      {
        "id": "e8dd54a7a8a9ebfe69f1ed1a9074fb3b2ceb078da4234b01c3b5da8168f78f7e",
        "domain": "omni",
        "question": "Call a polygon normal if it can be inscribed in a unit circle. How many non-congruent normal polygons are there such that the square of each side length is a positive integer?",
        "answer": "14",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2190,
        "difficulty": 4.0
      },
      {
        "id": "32c4708274dd2c1c174d3d46da59b3011f37131c528ade8e8d07b060feadb6e0",
        "domain": "omni",
        "question": "Let $a_{0}, a_{1}, a_{2}, \\ldots$ be a sequence of real numbers defined by $a_{0}=21, a_{1}=35$, and $a_{n+2}=4 a_{n+1}-4 a_{n}+n^{2}$ for $n \\geq 2$. Compute the remainder obtained when $a_{2006}$ is divided by 100.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 574,
        "difficulty": 5.0
      },
      {
        "id": "ef0a4057f4360333168d3355ef1c0a240a643ba265be2576823ede0000562b55",
        "domain": "omni",
        "question": "At the round table, $10$ people are sitting, some of them are knights, and the rest are liars (knights always say pride, and liars always lie) .  It is clear thath I have at least one knight and at least one liar. \nWhat is the largest number of those sitting at the table can say: ''Both of my neighbors are knights '' ?\n(A statement that is at least partially false is considered false.)",
        "answer": "9",
        "topic": "Mathematics -> Discrete Mathematics -> Logic",
        "source": "caucasus_mathematical_olympiad",
        "source_index": 3560,
        "difficulty": 4.0
      },
      {
        "id": "544a600f771f83e524ea0380136e7c192737cc1071db4a6b1bc2a4ebfaadd7a5",
        "domain": "omni",
        "question": "For positive integers $n$, let the numbers $c(n)$ be determined by the rules $c(1) = 1$, $c(2n) = c(n)$, and $c(2n+1) = (-1)^n c(n)$. Find the value of \\[ \\sum_{n=1}^{2013} c(n) c(n+2). \\]",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Other",
        "source": "putnam",
        "source_index": 3240,
        "difficulty": 8.0
      },
      {
        "id": "56bea47c8faba905541535e9e970824e18c2fdf107ad6b704bfaa97702b9b905",
        "domain": "omni",
        "question": "Peter has $2022$ pieces of magnetic railroad cars, which are of two types: some have the front with north and the rear with south magnetic polarity, and some have the rear with north and the rear with south magnetic polarity (on these railroad cars the front and the rear can be distinguished). Peter wants to decide whether there is the same number of both types of cars. He can try to fit together two cars in one try. What is the least number of tries needed?",
        "answer": "2021",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "problems_from_the_kmal_magazine",
        "source_index": 3995,
        "difficulty": 4.0
      },
      {
        "id": "9e44583887c5a365afd5739c017f8ad8f5f2c297dcae14a6ee6b2a8ad4882df3",
        "domain": "omni",
        "question": "A circle passes through the points $(2,0)$ and $(4,0)$ and is tangent to the line $y=x$. Find the sum of all possible values for the $y$-coordinate of the center of the circle.",
        "answer": "-6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 1718,
        "difficulty": 4.5
      },
      {
        "id": "bc6e07b4b566aa918d73f1a4fc0f5ea34ac03cf9a2a030d2f72dbf18ed146fc1",
        "domain": "omni",
        "question": "Let $ABC$ be a triangle with $AB=5, BC=4$ and $AC=3$. Let $\\mathcal{P}$ and $\\mathcal{Q}$ be squares inside $ABC$ with disjoint interiors such that they both have one side lying on $AB$. Also, the two squares each have an edge lying on a common line perpendicular to $AB$, and $\\mathcal{P}$ has one vertex on $AC$ and $\\mathcal{Q}$ has one vertex on $BC$. Determine the minimum value of the sum of the areas of the two squares.",
        "answer": "\\frac{144}{49}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 487,
        "difficulty": 4.5
      },
      {
        "id": "cf5721cdcbd5f7d874a337f1ada317b1780aa9a01f365509341cd96392216656",
        "domain": "omni",
        "question": "If $\\frac{1}{6} + \\frac{1}{3} = \\frac{1}{x}$, what is the value of $x$?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2729,
        "difficulty": 1.5
      },
      {
        "id": "64218fbe1d6989c412ed0966002c6f192c5557a2755f759de0c69d2ea8e93776",
        "domain": "omni",
        "question": "Through a point in the interior of a triangle $A B C$, three lines are drawn, one parallel to each side. These lines divide the sides of the triangle into three regions each. Let $a, b$, and $c$ be the lengths of the sides opposite $\\angle A, \\angle B$, and $\\angle C$, respectively, and let $a^{\\prime}, b^{\\prime}$, and $c^{\\prime}$ be the lengths of the middle regions of the sides opposite $\\angle A, \\angle B$, and $\\angle C$, respectively. Find the numerical value of $a^{\\prime} / a+b^{\\prime} / b+c^{\\prime} / c$.",
        "answer": "1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 387,
        "difficulty": 4.5
      },
      {
        "id": "33683e0ceb6d5da049aa35223b28f5143905bd35dd19f01a6058a64afbf13fd3",
        "domain": "omni",
        "question": "You are trapped in a room with only one exit, a long hallway with a series of doors and land mines. To get out you must open all the doors and disarm all the mines. In the room is a panel with 3 buttons, which conveniently contains an instruction manual. The red button arms a mine, the yellow button disarms two mines and closes a door, and the green button opens two doors. Initially 3 doors are closed and 3 mines are armed. The manual warns that attempting to disarm two mines or open two doors when only one is armed/closed will reset the system to its initial state. What is the minimum number of buttons you must push to get out?",
        "answer": "9",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1385,
        "difficulty": 4.5
      },
      {
        "id": "7656197974f07989bf52640de1fdd2951470ce270fe090505c39a6808ae88482",
        "domain": "omni",
        "question": "A polygon \\(\\mathcal{P}\\) is drawn on the 2D coordinate plane. Each side of \\(\\mathcal{P}\\) is either parallel to the \\(x\\) axis or the \\(y\\) axis (the vertices of \\(\\mathcal{P}\\) do not have to be lattice points). Given that the interior of \\(\\mathcal{P}\\) includes the interior of the circle \\(x^{2}+y^{2}=2022\\), find the minimum possible perimeter of \\(\\mathcal{P}\\).",
        "answer": "8 \\sqrt{2022}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2342,
        "difficulty": 5.0
      },
      {
        "id": "bdd36011371f784d1923fa2b107a469d7a54534dad01f02ea4da1709e1d74621",
        "domain": "omni",
        "question": "The set of points $\\left(x_{1}, x_{2}, x_{3}, x_{4}\\right)$ in $\\mathbf{R}^{4}$ such that $x_{1} \\geq x_{2} \\geq x_{3} \\geq x_{4}$ is a cone (or hypercone, if you insist). Into how many regions is this cone sliced by the hyperplanes $x_{i}-x_{j}=1$ for $1 \\leq i<j \\leq n$ ?",
        "answer": "14",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 503,
        "difficulty": 5.5
      },
      {
        "id": "4b622a3b64c288d4ba3bf4605f66e516b5c99ed11b425bbbfe36aa803bbf7da3",
        "domain": "omni",
        "question": "An HMMT party has $m$ MIT students and $h$ Harvard students for some positive integers $m$ and $h$, For every pair of people at the party, they are either friends or enemies. If every MIT student has 16 MIT friends and 8 Harvard friends, and every Harvard student has 7 MIT enemies and 10 Harvard enemies, compute how many pairs of friends there are at the party.",
        "answer": "342",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1797,
        "difficulty": 5.0
      },
      {
        "id": "87b504a4ba7c71c4a373b66db47786abf142417fd6d76466e358f30b10fa023f",
        "domain": "omni",
        "question": "What is the sum of all of the possibilities for Sam's number if Sam thinks of a 5-digit number, Sam's friend Sally tries to guess his number, Sam writes the number of matching digits beside each of Sally's guesses, and a digit is considered \"matching\" when it is the correct digit in the correct position?",
        "answer": "526758",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "pascal",
        "source_index": 2980,
        "difficulty": 1.5
      },
      {
        "id": "a3aa5637ccc364decb1c7d9c6fc6b27b33f05dd1fda1be099384cedeb4c52eb9",
        "domain": "omni",
        "question": "In Middle-Earth, nine cities form a 3 by 3 grid. The top left city is the capital of Gondor and the bottom right city is the capital of Mordor. How many ways can the remaining cities be divided among the two nations such that all cities in a country can be reached from its capital via the grid-lines without passing through a city of the other country?",
        "answer": "30",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2194,
        "difficulty": 4.0
      },
      {
        "id": "a0ca830c1d2f8e6049ca8b16c8e1d75bfae089f3a71c49a59911064ffce55fd3",
        "domain": "omni",
        "question": "Many of the students in M. Gamache's class brought a skateboard or a bicycle to school yesterday. The ratio of the number of skateboards to the number of bicycles was $7:4$. There were 12 more skateboards than bicycles. How many skateboards and bicycles were there in total?",
        "answer": "44",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2932,
        "difficulty": 1.5
      },
      {
        "id": "8e66bec06eb0c56dfc4d5c57d574117c790ff05f2f5bdd9760dcfe9957f1fef3",
        "domain": "omni",
        "question": "What is 25% of 60?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 3279,
        "difficulty": 1.0
      },
      {
        "id": "b70680fdadfa69dd4312da024918f24b0267f6164b7c900303309b049ecc5605",
        "domain": "omni",
        "question": "A Vandal and a Moderator are editing a Wikipedia article. The article originally is error-free. Each day, the Vandal introduces one new error into the Wikipedia article. At the end of the day, the moderator checks the article and has a $2 / 3$ chance of catching each individual error still in the article. After 3 days, what is the probability that the article is error-free?",
        "answer": "\\frac{416}{729}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 556,
        "difficulty": 5.0
      },
      {
        "id": "b5fc12eed59e659f0817fc21416993e3787bc99ee4b5afa278b7e539779dcbc6",
        "domain": "omni",
        "question": "Let $S$ be the set of $10$-tuples of non-negative integers that have sum $2019$. For any tuple in $S$, if one of the numbers in the tuple is $\\geq 9$, then we can subtract $9$ from it, and add $1$ to the remaining numbers in the tuple. Call thus one operation. If for $A,B\\in S$ we can get from $A$ to $B$ in finitely many operations, then denote $A\\rightarrow B$.\n\n(1) Find the smallest integer $k$, such that if the minimum number in $A,B\\in S$ respectively are both $\\geq k$, then $A\\rightarrow B$ implies $B\\rightarrow A$.\n\n(2) For the $k$ obtained in (1), how many tuples can we pick from $S$, such that any two of these tuples $A,B$ that are distinct, $A\\not\\rightarrow B$.",
        "answer": "10^8",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 223,
        "difficulty": 8.0
      },
      {
        "id": "cdc62203b94f85842139368bc50e2984f43627f2e340e77e4f30f813d63390ec",
        "domain": "omni",
        "question": "What is the smallest integer that can be placed in the box so that $\\frac{1}{2} < \\frac{\\square}{9}$?",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 2926,
        "difficulty": 1.5
      },
      {
        "id": "ff310eef48c8e4617248cdeab63b6c0d044582f7a78a6b726f258b1506f98c8f",
        "domain": "omni",
        "question": "Acute triangle $A B C$ has circumcenter $O$. The bisector of $\\angle A B C$ and the altitude from $C$ to side $A B$ intersect at $X$. Suppose that there is a circle passing through $B, O, X$, and $C$. If $\\angle B A C=n^{\\circ}$, where $n$ is a positive integer, compute the largest possible value of $n$.",
        "answer": "67",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1610,
        "difficulty": 5.25
      },
      {
        "id": "6e596db790722d8ca461fe33fff26926fca949389933589b42bc1ef329d81807",
        "domain": "omni",
        "question": "Glen, Hao, Ioana, Julia, Karla, and Levi participated in the 2023 Canadian Team Mathematics Contest. On their team uniforms, each had a different number chosen from the list $11,12,13,14,15,16$. Hao's and Julia's numbers were even. Karla's and Levi's numbers were prime numbers. Glen's number was a perfect square. What was Ioana's number?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3494,
        "difficulty": 2.0
      },
      {
        "id": "5e544b0ae95722c749e46a1e999fe0f0540f38c76acd39f93bb7e426b3428008",
        "domain": "omni",
        "question": "Find the value of $$ \\binom{2003}{1}+\\binom{2003}{4}+\\binom{2003}{7}+\\cdots+\\binom{2003}{2002} $$",
        "answer": "\\left(2^{2003}-2\\right) / 3",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_2",
        "source_index": 677,
        "difficulty": 5.0
      },
      {
        "id": "7d126e400c7dc03c415c3a2c5e8b576f1879e9c92aefb4f7f5125d89e90c7398",
        "domain": "omni",
        "question": "Calculate $\\sum_{n=1}^{2001} n^{3}$.",
        "answer": "4012013006001",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 398,
        "difficulty": 5.25
      },
      {
        "id": "318b0a6de43cba9fbd23382bdd154faca3ebb77bca72a579f4d120645f1c4fdc",
        "domain": "omni",
        "question": "$100$ children stand in a line each having $100$ candies. In one move, one of them may take some of their candies and distribute them to a non-empty set of the remaining children. After what least number of moves can it happen that no two children have the same number of candies?",
        "answer": "30",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "ToT",
        "source_index": 4353,
        "difficulty": 6.5
      },
      {
        "id": "0b012bf0ad8ab2d764d3978f79d450727cf64cdb0b568979d51833bc55184aa8",
        "domain": "omni",
        "question": "In 2019, a team, including professor Andrew Sutherland of MIT, found three cubes of integers which sum to 42: $42=\\left(-8053873881207597 \\_\\right)^{3}+(80435758145817515)^{3}+(12602123297335631)^{3}$. One of the digits, labeled by an underscore, is missing. What is that digit?",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 1857,
        "difficulty": 3.5
      },
      {
        "id": "52a1199ac38d688b3f3c8320dce69e99e67ad7f0e118796eac22df36b678b09a",
        "domain": "omni",
        "question": "Anna thinks of an integer that is not a multiple of three, not a perfect square, and the sum of its digits is a prime number. What could the integer be?",
        "answer": "14",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 3488,
        "difficulty": 1.5
      },
      {
        "id": "33427fbfe406fd8de7ff010620e8ff355bb8b1895c047bee7db8cadf8950e8a5",
        "domain": "omni",
        "question": "How much money does Roman give Dale if Roman wins a contest with a prize of $\\$ 200$, gives $30 \\%$ of the prize to Jackie, and then splits $15 \\%$ of what remains equally between Dale and Natalia?",
        "answer": "\\$ 10.50",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3499,
        "difficulty": 1.5
      },
      {
        "id": "422270b571f0cc52343ad2456f3e8cf35a4f0edafc585c420c3c2e0937b56247",
        "domain": "omni",
        "question": "Let $x<0.1$ be a positive real number. Let the foury series be $4+4 x+4 x^{2}+4 x^{3}+\\ldots$, and let the fourier series be $4+44 x+444 x^{2}+4444 x^{3}+\\ldots$ Suppose that the sum of the fourier series is four times the sum of the foury series. Compute $x$.",
        "answer": "\\frac{3}{40}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Calculus -> Differential Calculus -> Series -> Other",
        "source": "HMMT_11",
        "source_index": 2503,
        "difficulty": 4.0
      },
      {
        "id": "50becb8134ebb4206d30e3e32a5888655170f4781f39e46dceae545e761a8e31",
        "domain": "omni",
        "question": "I have five different pairs of socks. Every day for five days, I pick two socks at random without replacement to wear for the day. Find the probability that I wear matching socks on both the third day and the fifth day.",
        "answer": "\\frac{1}{63}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2445,
        "difficulty": 4.0
      },
      {
        "id": "7920a3aa5c38dfbd6299d923d82beca71fca3ca9a72e252c4f8b7c3d9814bbbb",
        "domain": "omni",
        "question": "Let $s(n)$ denote the number of 1's in the binary representation of $n$. Compute $$\\frac{1}{255} \\sum_{0 \\leq n<16} 2^{n}(-1)^{s(n)}$$",
        "answer": "45",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1062,
        "difficulty": 2.0
      },
      {
        "id": "95de3f7f3cb540aecb3016e6caf6fb61a624ca6e0ff273870db45d6c407cda0c",
        "domain": "omni",
        "question": "Tessa has a figure created by adding a semicircle of radius 1 on each side of an equilateral triangle with side length 2, with semicircles oriented outwards. She then marks two points on the boundary of the figure. What is the greatest possible distance between the two points?",
        "answer": "3",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 806,
        "difficulty": 5.0
      },
      {
        "id": "3c198184a82fc6ea5c146b6008207f04be95aa45142963e1536505284f4aae1c",
        "domain": "omni",
        "question": "Determine the greatest possible value of \\(\\sum_{i=1}^{10} \\cos(3x_i)\\) for real numbers $x_1,x_2,\\dots,x_{10}$ satisfying \\(\\sum_{i=1}^{10} \\cos(x_i) = 0\\).",
        "answer": "\\frac{480}{49}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "putnam",
        "source_index": 3204,
        "difficulty": 8.0
      },
      {
        "id": "3827b5247192df6c02986e67db85e76c86bbdfdb5073961fdfed6a61d8820721",
        "domain": "omni",
        "question": "A sequence of positive integers is defined by $a_{0}=1$ and $a_{n+1}=a_{n}^{2}+1$ for each $n \\geq 0$. Find $\\operatorname{gcd}(a_{999}, a_{2004})$.",
        "answer": "677",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1285,
        "difficulty": 5.0
      },
      {
        "id": "037af9846385740d240d3eb9a47413caa4b0ed39735504003fab07225e3f862c",
        "domain": "omni",
        "question": "Let $r_{1}, r_{2}, r_{3}, r_{4}$ be the four roots of the polynomial $x^{4}-4 x^{3}+8 x^{2}-7 x+3$. Find the value of $\\frac{r_{1}^{2}}{r_{2}^{2}+r_{3}^{2}+r_{4}^{2}}+\\frac{r_{2}^{2}}{r_{1}^{2}+r_{3}^{2}+r_{4}^{2}}+\\frac{r_{3}^{2}}{r_{1}^{2}+r_{2}^{2}+r_{4}^{2}}+\\frac{r_{4}^{2}}{r_{1}^{2}+r_{2}^{2}+r_{3}^{2}}$",
        "answer": "-4",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2205,
        "difficulty": 4.5
      },
      {
        "id": "099dd99f9d0abd6010cfab6cba4cfc1f71977792b54f4d848f5e0234a7d626df",
        "domain": "omni",
        "question": "Find the number of nonempty sets $\\mathcal{F}$ of subsets of the set $\\{1, \\ldots, 2014\\}$ such that: (a) For any subsets $S_{1}, S_{2} \\in \\mathcal{F}, S_{1} \\cap S_{2} \\in \\mathcal{F}$. (b) If $S \\in \\mathcal{F}, T \\subseteq\\{1, \\ldots, 2014\\}$, and $S \\subseteq T$, then $T \\in \\mathcal{F}$.",
        "answer": "2^{2014}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 879,
        "difficulty": 5.0
      },
      {
        "id": "bad8d3770a124b9c2953c6082d26236a011540eb98df2205ccd012606909cb1a",
        "domain": "omni",
        "question": "Brave NiuNiu (a milk drink company) organizes a promotion during the Chinese New Year: one gets a red packet when buying a carton of milk of their brand, and there is one of the following characters in the red packet \"虎\"(Tiger), \"生\"(Gain), \"威\"(Strength). If one collects two \"虎\", one \"生\" and one \"威\", then they form a Chinese phrases \"虎虎生威\" (Pronunciation: hu hu sheng wei), which means \"Have the courage and strength of the tiger\". This is a nice blessing because the Chinese zodiac sign for the year 2022 is tiger. Soon, the product of Brave NiuNiu becomes quite popular and people hope to get a collection of \"虎虎生威\". Suppose that the characters in every packet are independently random, and each character has probability $\\frac{1}{3}$. What is the expectation of cartons of milk to collect \"虎虎生威\" (i.e. one collects at least 2 copies of \"虎\", 1 copy of \"生\", 1 copy of \"威\")? Options: (A) $6 \\frac{1}{3}$, (B) $7 \\frac{1}{3}$, (C) $8 \\frac{1}{3}$, (D) $9 \\frac{1}{3}$, (E) None of the above.",
        "answer": "7 \\frac{1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "alibaba_global_contest",
        "source_index": 2307,
        "difficulty": 7.0
      },
      {
        "id": "b1e00bdd2850690c465c2afb8f097cd50e3de81dc4c2c891c92fd89c2f114071",
        "domain": "omni",
        "question": "The mean (average) of 5 consecutive integers is 9. What is the smallest of these 5 integers?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2708,
        "difficulty": 1.5
      },
      {
        "id": "926169c0734df46f8d9536e82eae080908f9550b43f0a62f1f65b714594da249",
        "domain": "omni",
        "question": "Find the area of triangle $QCD$ given that $Q$ is the intersection of the line through $B$ and the midpoint of $AC$ with the plane through $A, C, D$ and $N$ is the midpoint of $CD$.",
        "answer": "\\frac{3 \\sqrt{3}}{20}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2434,
        "difficulty": 3.5
      },
      {
        "id": "8f9113b79c8b0c41c3551d4b4937ccbc7d2111466472fdc386a086f7bb586295",
        "domain": "omni",
        "question": "What is the perimeter of the shaded region in a \\( 3 \\times 3 \\) grid where some \\( 1 \\times 1 \\) squares are shaded?",
        "answer": "10",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "cayley",
        "source_index": 3277,
        "difficulty": 1.0
      },
      {
        "id": "b3aca11714637989505840b3be0966a07f903de6d2d57f53a601e3b9d44a294f",
        "domain": "omni",
        "question": "Let $a,b,c,d$ be real numbers such that $a^2+b^2+c^2+d^2=1$. Determine the minimum value of $(a-b)(b-c)(c-d)(d-a)$ and determine all values of $(a,b,c,d)$ such that the minimum value is achived.",
        "answer": "-\\frac{1}{8}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "apmo",
        "source_index": 3806,
        "difficulty": 7.0
      },
      {
        "id": "ee7d3230afa174c99cf891dc889485eb0adf83991d8e869a088404b43e6945ce",
        "domain": "omni",
        "question": "How many pairs of real numbers $(x, y)$ satisfy the equation $y^{4}-y^{2}=x y^{3}-x y=x^{3} y-x y=x^{4}-x^{2}=0$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2280,
        "difficulty": 4.0
      },
      {
        "id": "2affb3ef0c7376ded2c8f15c54ddfdaefb979949ec2b6f45760d3e24b8df480a",
        "domain": "omni",
        "question": "How many positive integers $n \\leq 2009$ have the property that $\\left\\lfloor\\log _{2}(n)\\right\\rfloor$ is odd?",
        "answer": "682",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 1577,
        "difficulty": 5.0
      },
      {
        "id": "00a0854d78e9cf053be97e1a7fde4379859f9366e24413166b57abaf5d1752a2",
        "domain": "omni",
        "question": "On a party with 99 guests, hosts Ann and Bob play a game (the hosts are not regarded as guests). There are 99 chairs arranged in a circle; initially, all guests hang around those chairs. The hosts take turns alternately. By a turn, a host orders any standing guest to sit on an unoccupied chair $c$. If some chair adjacent to $c$ is already occupied, the same host orders one guest on such chair to stand up (if both chairs adjacent to $c$ are occupied, the host chooses exactly one of them). All orders are carried out immediately. Ann makes the first move; her goal is to fulfill, after some move of hers, that at least $k$ chairs are occupied. Determine the largest $k$ for which Ann can reach the goal, regardless of Bob's play.",
        "answer": "34",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "izho",
        "source_index": 1833,
        "difficulty": 7.0
      },
      {
        "id": "a1e164379226bb58b32b4356adf369b2952b7bbb339844f567ab69448368e210",
        "domain": "omni",
        "question": "Find the largest integer $n$ such that $3^{512}-1$ is divisible by $2^{n}$.",
        "answer": "11",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 682,
        "difficulty": 5.0
      },
      {
        "id": "ef48a8201df6d7a5b64657b4dc9e46db7bd7d9244a1aee6d8e803ef675a1e42f",
        "domain": "omni",
        "question": "$H O W, B O W$, and $D A H$ are equilateral triangles in a plane such that $W O=7$ and $A H=2$. Given that $D, A, B$ are collinear in that order, find the length of $B A$.",
        "answer": "11",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2560,
        "difficulty": 4.0
      },
      {
        "id": "366073e47e2c5f61fb6bf5216005264d5cbf67dcb66d81cf1eadf882e85a4295",
        "domain": "omni",
        "question": "Consider a $7 \\times 7$ grid of squares. Let $f:\\{1,2,3,4,5,6,7\\} \\rightarrow\\{1,2,3,4,5,6,7\\}$ be a function; in other words, $f(1), f(2), \\ldots, f(7)$ are each (not necessarily distinct) integers from 1 to 7 . In the top row of the grid, the numbers from 1 to 7 are written in order; in every other square, $f(x)$ is written where $x$ is the number above the square. How many functions have the property that the bottom row is identical to the top row, and no other row is identical to the top row?",
        "answer": "1470",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations",
        "source": "HMMT_11",
        "source_index": 2213,
        "difficulty": 4.0
      },
      {
        "id": "5bfc9840d96c171f989488dfcbf36d0faf4255f3a1f4cd73a865d8a4669c8d2c",
        "domain": "omni",
        "question": "Pentagon $J A M E S$ is such that $A M=S J$ and the internal angles satisfy $\\angle J=\\angle A=\\angle E=90^{\\circ}$, and $\\angle M=\\angle S$. Given that there exists a diagonal of $J A M E S$ that bisects its area, find the ratio of the shortest side of $J A M E S$ to the longest side of $J A M E S$.",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 1899,
        "difficulty": 4.5
      },
      {
        "id": "b2137ce6764294890638ea182bef477eecb4d590236bd695f998f80357cfe2cc",
        "domain": "omni",
        "question": "Find the largest real $C$ such that for all pairwise distinct positive real $a_{1}, a_{2}, \\ldots, a_{2019}$ the following inequality holds $$\\frac{a_{1}}{\\left|a_{2}-a_{3}\\right|}+\\frac{a_{2}}{\\left|a_{3}-a_{4}\\right|}+\\ldots+\\frac{a_{2018}}{\\left|a_{2019}-a_{1}\\right|}+\\frac{a_{2019}}{\\left|a_{1}-a_{2}\\right|}>C$$",
        "answer": "1010",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "izho",
        "source_index": 1605,
        "difficulty": 8.0
      },
      {
        "id": "f60d6cd7427a50801e7311d1d7df8545e2806662ecf78e20809e268195608fca",
        "domain": "omni",
        "question": "The cells of a $8 \\times 8$ table are initially white. Alice and Bob play a game. First Alice paints $n$ of the fields in red. Then Bob chooses $4$ rows and $4$ columns from the table and paints all fields in them in black. Alice wins if there is at least one red field left. Find the least value of $n$ such that Alice can win the game no matter how Bob plays.",
        "answer": "13",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "jbmo_shortlist",
        "source_index": 3775,
        "difficulty": 7.0
      },
      {
        "id": "4fba139e90270b1f60f5e2354c902f51f475c3bc9c160f966ac3994281ed858c",
        "domain": "omni",
        "question": "Let $V=\\{1, \\ldots, 8\\}$. How many permutations $\\sigma: V \\rightarrow V$ are automorphisms of some tree?",
        "answer": "30212",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1203,
        "difficulty": 6.0
      },
      {
        "id": "421826b294c6a4b96e91c0756a885113ad3d2903fbe6382e9e1071b2f4a055cf",
        "domain": "omni",
        "question": "Let $S=\\{-100,-99,-98, \\ldots, 99,100\\}$. Choose a 50-element subset $T$ of $S$ at random. Find the expected number of elements of the set $\\{|x|: x \\in T\\}$.",
        "answer": "\\frac{8825}{201}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1187,
        "difficulty": 4.5
      },
      {
        "id": "1dedc16bc610db8312db547dad7151fe2983b30eef4ecc931d8c8d72b4964491",
        "domain": "omni",
        "question": "Let $(x, y)$ be a pair of real numbers satisfying $$56x+33y=\\frac{-y}{x^{2}+y^{2}}, \\quad \\text { and } \\quad 33x-56y=\\frac{x}{x^{2}+y^{2}}$$ Determine the value of $|x|+|y|$.",
        "answer": "\\frac{11}{65}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 515,
        "difficulty": 5.0
      },
      {
        "id": "e6e3e6d66161c4dfb936a0b768197c90f07b3c8edf145ef35d906a7acf0683bc",
        "domain": "omni",
        "question": "The integers $1,2,4,5,6,9,10,11,13$ are to be placed in the circles and squares below with one number in each shape. Each integer must be used exactly once and the integer in each circle must be equal to the sum of the integers in the two neighbouring squares. If the integer $x$ is placed in the leftmost square and the integer $y$ is placed in the rightmost square, what is the largest possible value of $x+y$?",
        "answer": "20",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "fermat",
        "source_index": 2845,
        "difficulty": 2.5
      },
      {
        "id": "8c653c76cc1d42005b659acf03bafb24cd4c81493f659b94f269acad2c9b1626",
        "domain": "omni",
        "question": "What is the value of $k$ if the side lengths of four squares are shown, and the area of the fifth square is $k$?",
        "answer": "36",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2968,
        "difficulty": 1.5
      },
      {
        "id": "e1a90a99d665e623fa70958b2902bebb38dfa1c73c6834e18c498ffcdbfbcfeb",
        "domain": "omni",
        "question": "What is the probability that a randomly selected set of 5 numbers from the set of the first 15 positive integers has a sum divisible by 3?",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 943,
        "difficulty": 4.0
      },
      {
        "id": "61a03d1493659f7633f6992e3d855730bd0846a5b557c66ba35e389b0b8c4b33",
        "domain": "omni",
        "question": "For a real number $x$, let $[x]$ be $x$ rounded to the nearest integer and $\\langle x\\rangle$ be $x$ rounded to the nearest tenth. Real numbers $a$ and $b$ satisfy $\\langle a\\rangle+[b]=98.6$ and $[a]+\\langle b\\rangle=99.3$. Compute the minimum possible value of $[10(a+b)]$.",
        "answer": "988",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "HMMT_2",
        "source_index": 1409,
        "difficulty": 5.0
      },
      {
        "id": "4731ae27a277e8d7569da66c38df9dbb72927b251b8ba1522fe7382fe423cffb",
        "domain": "omni",
        "question": "The set $S=\\{1,2,3, \\ldots, 49,50\\}$ contains the first 50 positive integers. After the multiples of 2 and the multiples of 3 are removed, how many integers remain in the set $S$?",
        "answer": "17",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 2914,
        "difficulty": 1.5
      },
      {
        "id": "84bd4ad356cba3e1a0e47bf0ad7aae6846914808d91b1b74bc3b4ff9d3ae9166",
        "domain": "omni",
        "question": "Let $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ be a function such that for any integers $x, y$, we have $f\\left(x^{2}-3 y^{2}\\right)+f\\left(x^{2}+y^{2}\\right)=2(x+y) f(x-y)$. Suppose that $f(n)>0$ for all $n>0$ and that $f(2015) \\cdot f(2016)$ is a perfect square. Find the minimum possible value of $f(1)+f(2)$.",
        "answer": "246",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "HMMT_2",
        "source_index": 837,
        "difficulty": 4.0
      },
      {
        "id": "8dd08e60b400959e5a57bc27a031d0a18b4736e93d330e258a3abd0160d74bb5",
        "domain": "omni",
        "question": "On the Cartesian grid, Johnny wants to travel from $(0,0)$ to $(5,1)$, and he wants to pass through all twelve points in the set $S=\\{(i, j) \\mid 0 \\leq i \\leq 1,0 \\leq j \\leq 5, i, j \\in \\mathbb{Z}\\}$. Each step, Johnny may go from one point in $S$ to another point in $S$ by a line segment connecting the two points. How many ways are there for Johnny to start at $(0,0)$ and end at $(5,1)$ so that he never crosses his own path?",
        "answer": "252",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1242,
        "difficulty": 4.0
      },
      {
        "id": "3b6856f50337862789734d91863880505f18177845f85d5f1b522344755e49b4",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=5, B C=4$, and $C A=3$. Initially, there is an ant at each vertex. The ants start walking at a rate of 1 unit per second, in the direction $A \\rightarrow B \\rightarrow C \\rightarrow A$ (so the ant starting at $A$ moves along ray $\\overrightarrow{A B}$, etc.). For a positive real number $t$ less than 3, let $A(t)$ be the area of the triangle whose vertices are the positions of the ants after $t$ seconds have elapsed. For what positive real number $t$ less than 3 is $A(t)$ minimized?",
        "answer": "\\frac{47}{24}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 3380,
        "difficulty": 4.0
      },
      {
        "id": "1a3631df171cbbfba665c786c4f66dcbaf6623d2e9c9f5c7bfa1aee900fd0af5",
        "domain": "omni",
        "question": "What is the value of the expression \\( 4 + \\frac{3}{10} + \\frac{9}{1000} \\)?",
        "answer": "4.309",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals",
        "source": "pascal",
        "source_index": 2927,
        "difficulty": 1.0
      },
      {
        "id": "ceef5ee9392c011d9eb2600c96715f3180cae60b164dfd01c29e273037e1efae",
        "domain": "omni",
        "question": "If $2^{x}=16$, what is the value of $2^{x+3}$?",
        "answer": "128",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3180,
        "difficulty": 2.0
      },
      {
        "id": "cef11add3abb0d4255f878c117c350c9756737b61d37de062c877e41356daa1d",
        "domain": "omni",
        "question": "The digits $1,2,3,4,5,6$ are randomly chosen (without replacement) to form the three-digit numbers $M=\\overline{A B C}$ and $N=\\overline{D E F}$. For example, we could have $M=413$ and $N=256$. Find the expected value of $M \\cdot N$.",
        "answer": "143745",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1971,
        "difficulty": 5.0
      },
      {
        "id": "703987e6ba2f6e62490bd82464eb523a1788a1c9bf7f866c6567bcf44340cc97",
        "domain": "omni",
        "question": "A rectangle has length 13 and width 10. The length and the width of the rectangle are each increased by 2. By how much does the area of the rectangle increase?",
        "answer": "50",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "pascal",
        "source_index": 3495,
        "difficulty": 1.5
      },
      {
        "id": "37bccce4643dd7b4217a643599aa021089b6571a28a27bcdba0ef5bcd8afee39",
        "domain": "omni",
        "question": "What is the value of $\frac{(20-16) \times (12+8)}{4}$?",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2805,
        "difficulty": 1.0
      },
      {
        "id": "2d4a38cc9ad8b8196a9b2e6812fc6afe772cec1826f9d8f21f6736dfcb3775c3",
        "domain": "omni",
        "question": "A solid rectangular prism has dimensions 4 by 2 by 2. A 1 by 1 by 1 cube is cut out of the corner creating the new solid shown. What is the surface area of the new solid?",
        "answer": "40",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Surface Area",
        "source": "cayley",
        "source_index": 2694,
        "difficulty": 2.0
      },
      {
        "id": "043fa96a629c4f562d556194e20f41ef4fe72553dbfc2fef7d51d305bc72e2e5",
        "domain": "omni",
        "question": "Mrs. Toad has a class of 2017 students, with unhappiness levels $1,2, \\ldots, 2017$ respectively. Today in class, there is a group project and Mrs. Toad wants to split the class in exactly 15 groups. The unhappiness level of a group is the average unhappiness of its members, and the unhappiness of the class is the sum of the unhappiness of all 15 groups. What's the minimum unhappiness of the class Mrs. Toad can achieve by splitting the class into 15 groups?",
        "answer": "1121",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 519,
        "difficulty": 5.0
      },
      {
        "id": "ce1248991652422dd8c063b94ad6d65bac76dd935e68ebe6eb1b2dbe851b4c62",
        "domain": "omni",
        "question": "Ten numbers have an average (mean) of 87. Two of those numbers are 51 and 99. What is the average of the other eight numbers?",
        "answer": "90",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2814,
        "difficulty": 1.5
      },
      {
        "id": "d424513580ecd1b4f04bdc9644ceb1881aaac7d53c3b0f829a4028163bfe1274",
        "domain": "omni",
        "question": "A math professor stands up in front of a room containing 100 very smart math students and says, 'Each of you has to write down an integer between 0 and 100, inclusive, to guess 'two-thirds of the average of all the responses.' Each student who guesses the highest integer that is not higher than two-thirds of the average of all responses will receive a prize.' If among all the students it is common knowledge that everyone will write down the best response, and there is no communication between students, what single integer should each of the 100 students write down?",
        "answer": "0",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 3329,
        "difficulty": 3.5
      },
      {
        "id": "9c748695c48eb1f227a5959f137ad175c77c46c068a62e44eac3c9c58faed957",
        "domain": "omni",
        "question": "We are given an infinite deck of cards, each with a real number on it. For every real number $x$, there is exactly one card in the deck that has $x$ written on it. Now two players draw disjoint sets $A$ and $B$ of $100$ cards each from this deck. We would like to define a rule that declares one of them a winner. This rule should satisfy the following conditions:\n   1. The winner only depends on the relative order of the $200$ cards: if the cards are laid down in increasing order face down and we are told which card belongs to which player, but not what numbers are written on them, we can still decide the winner.\n   2. If we write the elements of both sets in increasing order as $A =\\{ a_1 , a_2 , \\ldots, a_{100} \\}$ and $B= \\{ b_1 , b_2 , \\ldots , b_{100} \\}$, and $a_i > b_i$ for all $i$, then $A$ beats $B$.\n   3. If three players draw three disjoint sets $A, B, C$ from the deck, $A$ beats $B$ and $B$ beats $C$  then $A$ also beats $C$.\nHow many ways are there to define such a rule? Here, we consider two rules as different if there exist two sets $A$ and $B$ such that $A$ beats $B$ according to one rule, but $B$ beats $A$ according to the other.\n\n[i]",
        "answer": "100",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 3871,
        "difficulty": 9.0
      },
      {
        "id": "057bdda4cdab40859fe4f5c3a96f70c567c9a7392f92970bb0607e2623d519ca",
        "domain": "omni",
        "question": "Euler's Bridge: The following figure is the graph of the city of Konigsburg in 1736 - vertices represent sections of the cities, edges are bridges. An Eulerian path through the graph is a path which moves from vertex to vertex, crossing each edge exactly once. How many ways could World War II bombers have knocked out some of the bridges of Konigsburg such that the Allied victory parade could trace an Eulerian path through the graph? (The order in which the bridges are destroyed matters.)",
        "answer": "13023",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 1439,
        "difficulty": 5.0
      },
      {
        "id": "255bc2b09055f5a5de28ab1a07ab1e5f08e2fb764c5f7d2001574faf9480ceff",
        "domain": "omni",
        "question": "If $x \\%$ of 60 is 12, what is $15 \\%$ of $x$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2816,
        "difficulty": 1.5
      },
      {
        "id": "a2309456b547ff7b3c5faeb98bccbcabf2407cd0a4abc80d3dd3196cec9edd35",
        "domain": "omni",
        "question": "The cookies in a cookie jar contain a total of 100 raisins. All but one of the cookies are the same size and contain the same number of raisins. One cookie is larger and contains one more raisin than each of the others. The number of cookies in the jar is between 5 and 10, inclusive. How many raisins are in the larger cookie?",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3195,
        "difficulty": 2.5
      },
      {
        "id": "7172658ba0d1650f001b0d66eb0588d2842c5b1b28a8821726b485f0b346aa10",
        "domain": "omni",
        "question": "Alice and the Cheshire Cat play a game. At each step, Alice either (1) gives the cat a penny, which causes the cat to change the number of (magic) beans that Alice has from $n$ to $5n$ or (2) gives the cat a nickel, which causes the cat to give Alice another bean. Alice wins (and the cat disappears) as soon as the number of beans Alice has is greater than 2008 and has last two digits 42. What is the minimum number of cents Alice can spend to win the game, assuming she starts with 0 beans?",
        "answer": "35",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 550,
        "difficulty": 4.5
      },
      {
        "id": "378c15cdee269dee2bab993e76554dca6ba98439f58e72861b55a23738c3ba3b",
        "domain": "omni",
        "question": "Suppose that there are 16 variables $\\left\\{a_{i, j}\\right\\}_{0 \\leq i, j \\leq 3}$, each of which may be 0 or 1 . For how many settings of the variables $a_{i, j}$ do there exist positive reals $c_{i, j}$ such that the polynomial $$f(x, y)=\\sum_{0 \\leq i, j \\leq 3} a_{i, j} c_{i, j} x^{i} y^{j}$$ $(x, y \\in \\mathbb{R})$ is bounded below?",
        "answer": "126",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1381,
        "difficulty": 5.5
      },
      {
        "id": "7e93237ab29805b176ed104057d96f717f38152c7655bb378e06db67a9c98c38",
        "domain": "omni",
        "question": "The integer 636405 may be written as the product of three 2-digit positive integers. What is the sum of these three integers?",
        "answer": "259",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2931,
        "difficulty": 2.0
      },
      {
        "id": "800574cb648bde687a08ad9d2bc6f37b6451c19e197c751e917d7b13f5ec9d44",
        "domain": "omni",
        "question": "Compute the number of distinct pairs of the form (first three digits of $x$, first three digits of $x^{4}$ ) over all integers $x>10^{10}$. For example, one such pair is $(100,100)$ when $x=10^{10^{10}}$.",
        "answer": "4495",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2418,
        "difficulty": 4.0
      },
      {
        "id": "e4e9a1914a96e948025ea9a6be1eab25c0a22b8d148bdfdec7348a636ad071bb",
        "domain": "omni",
        "question": "In a game of rock-paper-scissors with $n$ people, the following rules are used to determine a champion: (a) In a round, each person who has not been eliminated randomly chooses one of rock, paper, or scissors to play. (b) If at least one person plays rock, at least one person plays paper, and at least one person plays scissors, then the round is declared a tie and no one is eliminated. If everyone makes the same move, then the round is also declared a tie. (c) If exactly two moves are represented, then everyone who made the losing move is eliminated from playing in all further rounds (for example, in a game with 8 people, if 5 people play rock and 3 people play scissors, then the 3 who played scissors are eliminated). (d) The rounds continue until only one person has not been eliminated. That person is declared the champion and the game ends. If a game begins with 4 people, what is the expected value of the number of rounds required for a champion to be determined?",
        "answer": "\\frac{45}{14}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2372,
        "difficulty": 4.5
      },
      {
        "id": "c917db0e32b24a4af4a296985795cc14fd3c866bc93dfc76acf61b883768c3e4",
        "domain": "omni",
        "question": "A convex polyhedron has $n$ faces that are all congruent triangles with angles $36^{\\circ}, 72^{\\circ}$, and $72^{\\circ}$. Determine, with proof, the maximum possible value of $n$.",
        "answer": "\\[ 36 \\]",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 318,
        "difficulty": 4.5
      },
      {
        "id": "923baaa0a6a6d953e6a87b2ae071db17b748824f1314d637cd49e73a1ad2eb25",
        "domain": "omni",
        "question": "Three real numbers \\( x, y, \\) and \\( z \\) are such that \\( (x+4) / 2=(y+9) /(z-3)=(x+5) /(z-5) \\). Determine the value of \\( x / y \\).",
        "answer": "1/2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1375,
        "difficulty": 5.5
      },
      {
        "id": "aa82178b32cc5acabfad03e3057788b087c18ed2bdb7184bbae1684a0ace0bdb",
        "domain": "omni",
        "question": "$A B C D$ is a regular tetrahedron of volume 1. Maria glues regular tetrahedra $A^{\\prime} B C D, A B^{\\prime} C D$, $A B C^{\\prime} D$, and $A B C D^{\\prime}$ to the faces of $A B C D$. What is the volume of the tetrahedron $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}$?",
        "answer": "\\frac{125}{27}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 1714,
        "difficulty": 5.0
      },
      {
        "id": "1efc95ef51da0a61637f4a81914530cb7b494298cf9d7b392e63064fbf46ffbc",
        "domain": "omni",
        "question": "Suppose that $a$ and $b$ are real numbers such that the line $y=a x+b$ intersects the graph of $y=x^{2}$ at two distinct points $A$ and $B$. If the coordinates of the midpoint of $A B$ are $(5,101)$, compute $a+b$.",
        "answer": "61",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "HMMT_11",
        "source_index": 2298,
        "difficulty": 4.0
      },
      {
        "id": "86a1cd93ea2422cd650abaad94d6265a7285b629ec3c05f5e05f71af0f4d34ba",
        "domain": "omni",
        "question": "Find the minimum positive integer $n\\ge 3$, such that there exist $n$ points $A_1,A_2,\\cdots, A_n$ satisfying no three points are collinear and for any $1\\le i\\le n$, there exist $1\\le j \\le n (j\\neq i)$, segment $A_jA_{j+1}$ pass through the midpoint of segment $A_iA_{i+1}$, where $A_{n+1}=A_1$",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "china_national_olympiad",
        "source_index": 295,
        "difficulty": 7.0
      },
      {
        "id": "6ea32560a774b8b3c43e77be3bddceb391ac862d7ac713793d6fe8dba71b91dd",
        "domain": "omni",
        "question": "A bug is on a corner of a cube. A healthy path for the bug is a path along the edges of the cube that starts and ends where the bug is located, uses no edge multiple times, and uses at most two of the edges adjacent to any particular face. Find the number of healthy paths.",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1647,
        "difficulty": 5.0
      },
      {
        "id": "1d602e7dbe8ccc1426da268a158e58f95f0ed7848658cd6e0b95e5a1157ddae3",
        "domain": "omni",
        "question": "Ellie's drawer of hair clips contains 4 red clips, 5 blue clips, and 7 green clips. Each morning, she randomly chooses one hair clip to wear for the day. She returns this clip to the drawer each evening. One morning, Kyne removes $k$ hair clips before Ellie can make her daily selection. As a result, the probability that Ellie chooses a red clip is doubled. What is a possible value of $k$?",
        "answer": "12",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "pascal",
        "source_index": 3055,
        "difficulty": 1.5
      },
      {
        "id": "44984e56a0fd420614cd7da882c91703f6f4ed9c916ecde5dca2fc405d387cc9",
        "domain": "omni",
        "question": "Suppose that a polynomial of the form $p(x)=x^{2010} \\pm x^{2009} \\pm \\cdots \\pm x \\pm 1$ has no real roots. What is the maximum possible number of coefficients of -1 in $p$?",
        "answer": "1005",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1363,
        "difficulty": 6.0
      },
      {
        "id": "d594e97b135502ec38753b46206d02dad9591d40d1daaf7c5434f496914eac99",
        "domain": "omni",
        "question": "There are 100 people in a room with ages $1,2, \\ldots, 100$. A pair of people is called cute if each of them is at least seven years older than half the age of the other person in the pair. At most how many pairwise disjoint cute pairs can be formed in this room?",
        "answer": "43",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1164,
        "difficulty": 5.0
      },
      {
        "id": "54bd4a69cc2740555840aac3e6d1977d340487c2ec70eccc7b3ea8fba3f05215",
        "domain": "omni",
        "question": "The $y$-intercepts of three parallel lines are 2, 3, and 4. The sum of the $x$-intercepts of the three lines is 36. What is the slope of these parallel lines?",
        "answer": "-\\frac{1}{4}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2892,
        "difficulty": 3.0
      },
      {
        "id": "b57a901ae9fc168d85c640185f5c4a8a929b919107d2fd75932b86b699634e8a",
        "domain": "omni",
        "question": "(Lucas Numbers) The Lucas numbers are defined by $L_{0}=2, L_{1}=1$, and $L_{n+2}=L_{n+1}+L_{n}$ for every $n \\geq 0$. There are $N$ integers $1 \\leq n \\leq 2016$ such that $L_{n}$ contains the digit 1 . Estimate $N$.",
        "answer": "1984",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_2",
        "source_index": 1075,
        "difficulty": 4.5
      },
      {
        "id": "a98192a0704c092636804f3426484656b86c7ed06dda0c8484b8060742d9a252",
        "domain": "omni",
        "question": "A subset $S$ of the set $\\{1,2, \\ldots, 10\\}$ is chosen randomly, with all possible subsets being equally likely. Compute the expected number of positive integers which divide the product of the elements of $S$. (By convention, the product of the elements of the empty set is 1.)",
        "answer": "\\frac{375}{8}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Mathematical Statistics",
        "source": "HMMT_2",
        "source_index": 1604,
        "difficulty": 4.5
      },
      {
        "id": "1123fcd556fc0ea4eece935b32bed72d1e9b35fc8379fcc048bd6767cd35d565",
        "domain": "omni",
        "question": "If $N$ is a positive integer between 1000000 and 10000000, inclusive, what is the maximum possible value for the sum of the digits of $25 \\times N$?",
        "answer": "67",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "pascal",
        "source_index": 3220,
        "difficulty": 2.5
      },
      {
        "id": "bba8aac9b3f36ae3fe5bcdd360710d3f6be1dadee20989e44f6ea852756468f9",
        "domain": "omni",
        "question": "If $a(x+2)+b(x+2)=60$ and $a+b=12$, what is the value of $x$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2819,
        "difficulty": 2.0
      },
      {
        "id": "a08c19322b1262ff84d28bf323f2446b6a023d061b1ee65ef152140e9231e919",
        "domain": "omni",
        "question": "Points $A, B, C$, and $D$ are on a line in that order. The distance from $A$ to $D$ is 24. The distance from $B$ to $D$ is 3 times the distance from $A$ to $B$. Point $C$ is halfway between $B$ and $D$. What is the distance from $A$ to $C$?",
        "answer": "15",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "cayley",
        "source_index": 2840,
        "difficulty": 2.0
      },
      {
        "id": "9889cfb168805a073b42cc1bdb99c356d4401fa21129d4f425b11d7757f32e8d",
        "domain": "omni",
        "question": "Let \\(ABCD\\) be an isosceles trapezoid with \\(AB=1, BC=DA=5, CD=7\\). Let \\(P\\) be the intersection of diagonals \\(AC\\) and \\(BD\\), and let \\(Q\\) be the foot of the altitude from \\(D\\) to \\(BC\\). Let \\(PQ\\) intersect \\(AB\\) at \\(R\\). Compute \\(\\sin \\angle RPD\\).",
        "answer": "\\frac{4}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2214,
        "difficulty": 5.0
      },
      {
        "id": "3e260a02b186976468c997dcdffa0f093e2a220a377967b9d4af75871bb0c23b",
        "domain": "omni",
        "question": "How many 5-digit numbers $\\overline{a b c d e}$ exist such that digits $b$ and $d$ are each the sum of the digits to their immediate left and right? (That is, $b=a+c$ and $d=c+e$.)",
        "answer": "330",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1304,
        "difficulty": 4.0
      },
      {
        "id": "7de9bce379632e4743362858b7670e95ff63ecc32f0d3064579809ef09e08d90",
        "domain": "omni",
        "question": "Let $A B C D E$ be a convex pentagon such that $\\angle A B C=\\angle A C D=\\angle A D E=90^{\\circ}$ and $A B=B C=C D=D E=1$. Compute $A E$.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 669,
        "difficulty": 3.5
      },
      {
        "id": "8d17a127d2c95cd5a2edfe686ac8d66a579ec9e24a5de825a39442e51c89c822",
        "domain": "omni",
        "question": "A line of soldiers 1 mile long is jogging. The drill sergeant, in a car, moving at twice their speed, repeatedly drives from the back of the line to the front of the line and back again. When each soldier has marched 15 miles, how much mileage has been added to the car, to the nearest mile?",
        "answer": "30",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 3308,
        "difficulty": 3.5
      },
      {
        "id": "34712a71a776ff76e9ac31de3a9a461176f68809f7569c3ebea3e13efe015a2c",
        "domain": "omni",
        "question": "Triangle $A B C$ is given in the plane. Let $A D$ be the angle bisector of $\\angle B A C$; let $B E$ be the altitude from $B$ to $A D$, and let $F$ be the midpoint of $A B$. Given that $A B=28, B C=33, C A=37$, what is the length of $E F$ ?",
        "answer": "14",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2440,
        "difficulty": 4.5
      },
      {
        "id": "efb64075ba7db4b4c869940440b85f935e6fd52f2b09bb17c8c4ef07736916fa",
        "domain": "omni",
        "question": "Let $f(x)$ be a quotient of two quadratic polynomials. Given that $f(n)=n^{3}$ for all $n \\in\\{1,2,3,4,5\\}$, compute $f(0)$.",
        "answer": "\\frac{24}{17}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1663,
        "difficulty": 6.0
      },
      {
        "id": "88a0e7ae2544be112b460ee87c30bb84b775b120af0e1d035041bdceefd99e48",
        "domain": "omni",
        "question": "Alison is eating 2401 grains of rice for lunch. She eats the rice in a very peculiar manner: every step, if she has only one grain of rice remaining, she eats it. Otherwise, she finds the smallest positive integer $d>1$ for which she can group the rice into equal groups of size $d$ with none left over. She then groups the rice into groups of size $d$, eats one grain from each group, and puts the rice back into a single pile. How many steps does it take her to finish all her rice?",
        "answer": "17",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2096,
        "difficulty": 4.0
      },
      {
        "id": "84aae0a89471176d3d50a3edf92b4b10cfc0387dcbbd65a5a2f7e6aed96f21f6",
        "domain": "omni",
        "question": "Chris and Paul each rent a different room of a hotel from rooms $1-60$. However, the hotel manager mistakes them for one person and gives \"Chris Paul\" a room with Chris's and Paul's room concatenated. For example, if Chris had 15 and Paul had 9, \"Chris Paul\" has 159. If there are 360 rooms in the hotel, what is the probability that \"Chris Paul\" has a valid room?",
        "answer": "\\frac{153}{1180}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1875,
        "difficulty": 4.75
      },
      {
        "id": "cf62d316d25e72f1466a06b23ce649cee055d4f58f0a9480892b2dd2ac861388",
        "domain": "omni",
        "question": "679 contestants participated in HMMT February 2017. Let \\(N\\) be the number of these contestants who performed at or above the median score in at least one of the three individual tests. Estimate \\(N\\). An estimate of \\(E\\) earns \\(\\left\\lfloor 20-\\frac{|E-N|}{2}\\right\\rfloor\\) or 0 points, whichever is greater.",
        "answer": "516",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Mathematical Statistics",
        "source": "HMMT_2",
        "source_index": 1575,
        "difficulty": 5.0
      },
      {
        "id": "f4c9cb79a1b938af1341738112f8f3d0bff100efac7b23af0a47f412c0faa71b",
        "domain": "omni",
        "question": "Let \\(a \\star b=\\sin a \\cos b\\) for all real numbers \\(a\\) and \\(b\\). If \\(x\\) and \\(y\\) are real numbers such that \\(x \\star y-y \\star x=1\\), what is the maximum value of \\(x \\star y+y \\star x\\)?",
        "answer": "1",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 351,
        "difficulty": 5.25
      },
      {
        "id": "0ef38ee9cd0f1082921e0b7d2b3d236839d4feb82e9146f168599aeae3d78af8",
        "domain": "omni",
        "question": "The squares of a $3 \\times 3$ grid are filled with positive integers such that 1 is the label of the upperleftmost square, 2009 is the label of the lower-rightmost square, and the label of each square divides the one directly to the right of it and the one directly below it. How many such labelings are possible?",
        "answer": "2448",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1067,
        "difficulty": 6.0
      },
      {
        "id": "fdefbaf7c249af0847154505a0e16d45e2974ee6a8f6883cf6b8c2d86242c5ab",
        "domain": "omni",
        "question": "Let the sequence $a_{i}$ be defined as $a_{i+1}=2^{a_{i}}$. Find the number of integers $1 \\leq n \\leq 1000$ such that if $a_{0}=n$, then 100 divides $a_{1000}-a_{1}$.",
        "answer": "50",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 2562,
        "difficulty": 4.0
      },
      {
        "id": "49f51ef0db0201d8aeec1527f8d9e6c1b76f33cd18b81d437fcf650b945c651c",
        "domain": "omni",
        "question": "Shanille O'Keal shoots free throws on a basketball court. She hits the first and misses the second, and thereafter the probability that she hits the next shot is equal to the proportion of shots she has hit so far. What is the probability she hits exactly 50 of her first 100 shots?",
        "answer": "\\(\\frac{1}{99}\\)",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "putnam",
        "source_index": 3243,
        "difficulty": 9.0
      },
      {
        "id": "0bf2a0b03e3b507e2cc402430cab8f05bccc8b7ca111f7815837ee3fdb159aee",
        "domain": "omni",
        "question": "If $n$ is a positive integer, let $s(n)$ denote the sum of the digits of $n$. We say that $n$ is zesty if there exist positive integers $x$ and $y$ greater than 1 such that $x y=n$ and $s(x) s(y)=s(n)$. How many zesty two-digit numbers are there?",
        "answer": "34",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 981,
        "difficulty": 3.5
      },
      {
        "id": "21d3737ca8b0335acf39c64c68f83a21645908a75f39c341c076e924df585318",
        "domain": "omni",
        "question": "What is the difference between the largest and smallest numbers in the list $0.023,0.302,0.203,0.320,0.032$?",
        "answer": "0.297",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals",
        "source": "pascal",
        "source_index": 3026,
        "difficulty": 1.0
      },
      {
        "id": "a9db00ff56274ee28480d2393d235c98e2ba73e62e3fd3375eb5a914054a98a6",
        "domain": "omni",
        "question": "Which of the following integers cannot be written as a product of two integers, each greater than 1: 6, 27, 53, 39, 77?",
        "answer": "53",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "fermat",
        "source_index": 3527,
        "difficulty": 1.5
      },
      {
        "id": "691c1453da8f2c1ba2b4768a9f49bf8e42f5bebc16c7c0e5bbbf4b23fc6eb655",
        "domain": "omni",
        "question": "The Antarctican language has an alphabet of just 16 letters. Interestingly, every word in the language has exactly 3 letters, and it is known that no word's first letter equals any word's last letter (for instance, if the alphabet were $\\{a, b\\}$ then $a a b$ and aaa could not both be words in the language because $a$ is the first letter of a word and the last letter of a word; in fact, just aaa alone couldn't be in the language). Given this, determine the maximum possible number of words in the language.",
        "answer": "1024",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 765,
        "difficulty": 4.5
      },
      {
        "id": "e268a092679c7ff04d9f8d635d417522eab839a9cd1f0fb02fca9c0b0656ec6e",
        "domain": "omni",
        "question": "Pick a random digit in the decimal expansion of $\\frac{1}{99999}$. What is the probability that it is 0?",
        "answer": "\\frac{4}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 1709,
        "difficulty": 3.5
      },
      {
        "id": "7ace41011613407f00b321f90e05a7f7d8f482ed8311eb3f61c86a986032b877",
        "domain": "omni",
        "question": "Let $z_{0}+z_{1}+z_{2}+\\cdots$ be an infinite complex geometric series such that $z_{0}=1$ and $z_{2013}=\\frac{1}{2013^{2013}}$. Find the sum of all possible sums of this series.",
        "answer": "\\frac{2013^{2014}}{2013^{2013}-1}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 601,
        "difficulty": 5.25
      },
      {
        "id": "62f8a847acf34a0fb809bd3d1fbc70cf34ce9f0c4c2f999cee14ae9adf0e4f34",
        "domain": "omni",
        "question": "The top section of an 8 cm by 6 cm rectangular sheet of paper is folded along a straight line so that when the top section lies flat on the bottom section, corner $P$ lies on top of corner $R$. What is the length of the crease?",
        "answer": "7.5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "cayley",
        "source_index": 3109,
        "difficulty": 3.0
      },
      {
        "id": "86750b75d04b94f0142b7981843c6aa08eb6e12a0d987bd7416b4138680eb580",
        "domain": "omni",
        "question": "Assume the quartic $x^{4}-a x^{3}+b x^{2}-a x+d=0$ has four real roots $\\frac{1}{2} \\leq x_{1}, x_{2}, x_{3}, x_{4} \\leq 2$. Find the maximum possible value of $\\frac{\\left(x_{1}+x_{2}\\right)\\left(x_{1}+x_{3}\\right) x_{4}}{\\left(x_{4}+x_{2}\\right)\\left(x_{4}+x_{3}\\right) x_{1}}$ (over all valid choices of $\\left.a, b, d\\right)$.",
        "answer": "\\frac{5}{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1686,
        "difficulty": 5.0
      },
      {
        "id": "56b072b94aade7b002bba6cb2a67fe7044be3b395366843ea97cb7398c15955a",
        "domain": "omni",
        "question": "Let $V$ be a rectangular prism with integer side lengths. The largest face has area 240 and the smallest face has area 48. A third face has area $x$, where $x$ is not equal to 48 or 240. What is the sum of all possible values of $x$?",
        "answer": "260",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2584,
        "difficulty": 4.0
      },
      {
        "id": "f782b2d926ba64c1981538639ce36fe68821f81195c4b7771ba21cc2a12fe523",
        "domain": "omni",
        "question": "The gas tank in Catherine's car is $\\frac{1}{8}$ full. When 30 litres of gas are added, the tank becomes $\\frac{3}{4}$ full. If the gas costs Catherine $\\$ 1.38$ per litre, how much will it cost her to fill the remaining quarter of the tank?",
        "answer": "\\$16.56",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3433,
        "difficulty": 1.5
      },
      {
        "id": "6914967c1e1ebf6292fcbb339355b88c4c854ffdd284d8cb3867fd6e650d99f9",
        "domain": "omni",
        "question": "How many ways, without taking order into consideration, can 2002 be expressed as the sum of 3 positive integers (for instance, $1000+1000+2$ and $1000+2+1000$ are considered to be the same way)?",
        "answer": "334000",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1089,
        "difficulty": 5.0
      },
      {
        "id": "5a11f158fe0e5dba8f8a1b23b3abc780e27565951ff08b92404a27e756c2bc7c",
        "domain": "omni",
        "question": "In triangle $A B C, \\angle A B C$ is obtuse. Point $D$ lies on side $A C$ such that \\angle A B D$ is right, and point $E$ lies on side $A C$ between $A$ and $D$ such that $B D$ bisects \\angle E B C$. Find $C E$, given that $A C=35, B C=7$, and $B E=5$.",
        "answer": "10",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1475,
        "difficulty": 5.0
      },
      {
        "id": "09b719eea3d656c5b278dae6b75db267a5344efc0d7db6c0957967bc59721daa",
        "domain": "omni",
        "question": "Consider the addition problem: \\begin{tabular}{ccccc} & C & A & S & H \\\\ + & & & M & E \\\\ \\hline O & S & I & D & E \\end{tabular} where each letter represents a base-ten digit, and $C, M, O \\neq 0$. (Distinct letters are allowed to represent the same digit) How many ways are there to assign values to the letters so that the addition problem is true?",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2241,
        "difficulty": 4.5
      },
      {
        "id": "395ea58a090060e20d293dd38a99a506e4bf020b56e8e0d45f4e3f9fa1b30a4f",
        "domain": "omni",
        "question": "Suppose that $a, b$ and $c$ are integers with $(x-a)(x-6)+3=(x+b)(x+c)$ for all real numbers $x$. What is the sum of all possible values of $b$?",
        "answer": "-24",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2903,
        "difficulty": 2.5
      },
      {
        "id": "542a8dbabf414f8be98fbce41c769ac25574ca4d8682ea0cb98577452a281e62",
        "domain": "omni",
        "question": "We call the pair $(m, n)$ of positive integers a happy pair if the greatest common divisor of $m$ and $n$ is a perfect square. For example, $(20, 24)$ is a happy pair because the greatest common divisor of 20 and 24 is 4. Suppose that $k$ is a positive integer such that $(205800, 35k)$ is a happy pair. What is the number of possible values of $k$ with $k \\leq 2940$?",
        "answer": "30",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "pascal",
        "source_index": 2993,
        "difficulty": 2.0
      },
      {
        "id": "3c65c41d6c4772878dd47bcdb5213c2f0e172e6ce8b1df923d45672c6eaed222",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, a_{3}, \\ldots$ be a sequence of positive integers where $a_{1}=\\sum_{i=0}^{100} i$! and $a_{i}+a_{i+1}$ is an odd perfect square for all $i \\geq 1$. Compute the smallest possible value of $a_{1000}$.",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2592,
        "difficulty": 5.0
      },
      {
        "id": "4f7537d641f00bea7577bf46146c272f2b8d5c177ddb936e3c2759a8e0d6d009",
        "domain": "omni",
        "question": "A positive integer is called [i]fancy[/i] if it can be expressed in the form $$2^{a_1}+2^{a_2}+ \\cdots+ 2^{a_{100}},$$ where $a_1,a_2, \\cdots, a_{100}$ are non-negative integers that are not necessarily distinct. Find the smallest positive integer $n$ such that no multiple of $n$ is a [i]fancy[/i] number.",
        "answer": "2^{101} - 1",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "apmo",
        "source_index": 3632,
        "difficulty": 7.0
      },
      {
        "id": "0f70b7c5b72310e9de2c50cbadc9f53c7735efbb402ddf258105663047c4640a",
        "domain": "omni",
        "question": "On a $3 \\times 3$ chessboard, each square contains a knight with $\\frac{1}{2}$ probability. What is the probability that there are two knights that can attack each other? (In chess, a knight can attack any piece which is two squares away from it in a particular direction and one square away in a perpendicular direction.)",
        "answer": "\\frac{209}{256}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2557,
        "difficulty": 4.0
      },
      {
        "id": "238c6aeb35da678271620f901dd5007a999f88bd703c7d37d74807c91a9d37c4",
        "domain": "omni",
        "question": "Given $30$ students such that each student has at most $5$ friends and for every $5$ students there is a pair of students that are not friends, determine the maximum $k$ such that for all such possible configurations, there exists $k$ students who are all not friends.",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "china_national_olympiad",
        "source_index": 10,
        "difficulty": 7.0
      },
      {
        "id": "deb7565fccbbd2c652d63c8bf42c28d13bfd0e1bbebc35bd93ef32e96e520f0f",
        "domain": "omni",
        "question": "Bernie has 2020 marbles and 2020 bags labeled $B_{1}, \\ldots, B_{2020}$ in which he randomly distributes the marbles (each marble is placed in a random bag independently). If $E$ the expected number of integers $1 \\leq i \\leq 2020$ such that $B_{i}$ has at least $i$ marbles, compute the closest integer to $1000E$.",
        "answer": "1000",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2509,
        "difficulty": 4.0
      },
      {
        "id": "722167b70beeee97bbe94e7c8addb18abd429daf4435a2af8549e6e03dee327b",
        "domain": "omni",
        "question": "Find the maximum possible number of diagonals of equal length in a convex hexagon.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 568,
        "difficulty": 4.5
      },
      {
        "id": "66e9fd3e1ffea5e35df78c0ceb24e2a5a55f04efacb62b5dd11a1b4a59689264",
        "domain": "omni",
        "question": "Let $n$ be the second smallest integer that can be written as the sum of two positive cubes in two different ways. Compute $n$.",
        "answer": "4104",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1767,
        "difficulty": 4.5
      },
      {
        "id": "9f3e48d254203c8caca44390bcb4610e7efc812cc4e89998dd6bb7b10640243f",
        "domain": "omni",
        "question": "What fraction of the area of a regular hexagon of side length 1 is within distance $\\frac{1}{2}$ of at least one of the vertices?",
        "answer": "\\pi \\sqrt{3} / 9",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 3313,
        "difficulty": 4.5
      },
      {
        "id": "828e270dbc421e4f97648464d82d8954a914d9a0d32a0606a58a7a3e7807d73c",
        "domain": "omni",
        "question": "What is the smallest number of regular hexagons of side length 1 needed to completely cover a disc of radius 1 ?",
        "answer": "3",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 424,
        "difficulty": 4.0
      },
      {
        "id": "264c6c89622aaaee747672b5be91d5507df3323cda44c4071ea58f22e921f3a0",
        "domain": "omni",
        "question": "Steph Curry is playing the following game and he wins if he has exactly 5 points at some time. Flip a fair coin. If heads, shoot a 3-point shot which is worth 3 points. If tails, shoot a free throw which is worth 1 point. He makes \\frac{1}{2} of his 3-point shots and all of his free throws. Find the probability he will win the game. (Note he keeps flipping the coin until he has exactly 5 or goes over 5 points)",
        "answer": "\\frac{140}{243}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2122,
        "difficulty": 4.0
      },
      {
        "id": "1843def12a20cceca137fd3b4f1cf208d67f45d096762e4c5afbf7151e964654",
        "domain": "omni",
        "question": "The UEFA Champions League playoffs is a 16-team soccer tournament in which Spanish teams always win against non-Spanish teams. In each of 4 rounds, each remaining team is randomly paired against one other team; the winner advances to the next round, and the loser is permanently knocked out of the tournament. If 3 of the 16 teams are Spanish, what is the probability that there are 2 Spanish teams in the final round?",
        "answer": "$\\frac{4}{5}$",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1973,
        "difficulty": 4.5
      },
      {
        "id": "d0a20bda982c289097df007279e3d5859cf92f593a38d4e888b8c3c8e5fb98cb",
        "domain": "omni",
        "question": "In a cafeteria line, the number of people ahead of Kaukab is equal to two times the number of people behind her. There are $n$ people in the line. What is a possible value of $n$?",
        "answer": "25",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2756,
        "difficulty": 1.5
      },
      {
        "id": "5cc38e30f0647dd0eb3db69b168c7dcdaf4bc4903b2b1b2bf6edffd759bb7293",
        "domain": "omni",
        "question": "On a blackboard a stranger writes the values of $s_{7}(n)^{2}$ for $n=0,1, \\ldots, 7^{20}-1$, where $s_{7}(n)$ denotes the sum of digits of $n$ in base 7 . Compute the average value of all the numbers on the board.",
        "answer": "3680",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2466,
        "difficulty": 4.0
      },
      {
        "id": "5701217280a04c0bcfd248ca087481082293efcde7dd3dde606cbe328a36de51",
        "domain": "omni",
        "question": "The numbers $1,2,\\ldots,64$ are written in the squares of an $8\\times 8$ chessboard, one number to each square. Then $2\\times 2$ tiles are placed on the chessboard (without overlapping) so that each tile covers exactly four squares whose numbers sum to less than $100$. Find, with proof, the maximum number of tiles that can be placed on the chessboard, and give an example of a distribution of the numbers $1,2,\\ldots,64$ into the squares of the chessboard that admits this maximum number of tiles.",
        "answer": "12",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "cono_sur_olympiad",
        "source_index": 3669,
        "difficulty": 6.0
      },
      {
        "id": "cb03d96996abfa1d801f0a7093e8db6be26c8146f4e8b640874d7669ce253001",
        "domain": "omni",
        "question": "The function $f: \\mathbb{Z}^{2} \\rightarrow \\mathbb{Z}$ satisfies - $f(x, 0)=f(0, y)=0$, and - $f(x, y)=f(x-1, y)+f(x, y-1)+x+y$ for all nonnegative integers $x$ and $y$. Find $f(6,12)$.",
        "answer": "77500",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2569,
        "difficulty": 4.0
      },
      {
        "id": "7fec22e3fe53531d7d51e1d96e36e57716436ee526fcc842342fab17d88b19dd",
        "domain": "omni",
        "question": "The train schedule in Hummut is hopelessly unreliable. Train A will enter Intersection X from the west at a random time between 9:00 am and 2:30 pm; each moment in that interval is equally likely. Train B will enter the same intersection from the north at a random time between 9:30 am and 12:30 pm, independent of Train A; again, each moment in the interval is equally likely. If each train takes 45 minutes to clear the intersection, what is the probability of a collision today?",
        "answer": "\\frac{13}{48}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1450,
        "difficulty": 4.5
      },
      {
        "id": "a9290c94445b3b90cadb9ef78d0f41de9e62767eb24ff150181679fe202392a8",
        "domain": "omni",
        "question": "What is the last digit of $1^{1}+2^{2}+3^{3}+\\cdots+100^{100}$?",
        "answer": "0",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1293,
        "difficulty": 4.0
      },
      {
        "id": "058ec5b1a48ffca5735f9ead2bb7c170ccc7fd59038596c2ce3e234665d88d35",
        "domain": "omni",
        "question": "For a finite graph $G$, let $f(G)$ be the number of triangles and $g(G)$ the number of tetrahedra formed by edges of $G$. Find the least constant $c$ such that \\[g(G)^3\\le c\\cdot f(G)^4\\] for every graph $G$.\n\n[i]",
        "answer": "\\frac{3}{32}",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "imo_shortlist",
        "source_index": 4074,
        "difficulty": 8.0
      },
      {
        "id": "990b4a821c92f3ca9f8b371b488a2815e47047663e7272b5b057bc09180d45bb",
        "domain": "omni",
        "question": "Suppose that $x$ and $y$ are real numbers that satisfy the two equations $3x+2y=6$ and $9x^2+4y^2=468$. What is the value of $xy$?",
        "answer": "-36",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2815,
        "difficulty": 2.5
      },
      {
        "id": "374a24ee2c36fe8a8e7be41a3ca68c489f627660c9f809c7665c7ad9cfe3aa7d",
        "domain": "omni",
        "question": "What is the probability that exactly one person gets their hat back when 6 people randomly pick hats?",
        "answer": "\\frac{11}{30}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 949,
        "difficulty": 3.5
      },
      {
        "id": "fcd11e3d366e10ea67d4d4f6e8d1aefae7b760041b3bbbec584345fed4855d90",
        "domain": "omni",
        "question": "The attached figure is an undirected graph. The circled numbers represent the nodes, and the numbers along the edges are their lengths (symmetrical in both directions). An Alibaba Hema Xiansheng carrier starts at point A and will pick up three orders from merchants B_{1}, B_{2}, B_{3} and deliver them to three customers C_{1}, C_{2}, C_{3}, respectively. The carrier drives a scooter with a trunk that holds at most two orders at any time. All the orders have equal size. Find the shortest travel route that starts at A and ends at the last delivery. To simplify this question, assume no waiting time during each pickup and delivery.",
        "answer": "16",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "alibaba_global_contest",
        "source_index": 1725,
        "difficulty": 4.5
      },
      {
        "id": "7607e7c496e908ecf2788a41641e359e0fa6ee2dee97162b61010aae61628b04",
        "domain": "omni",
        "question": "The classrooms at MIT are each identified with a positive integer (with no leading zeroes). One day, as President Reif walks down the Infinite Corridor, he notices that a digit zero on a room sign has fallen off. Let $N$ be the original number of the room, and let $M$ be the room number as shown on the sign. The smallest interval containing all possible values of $\\frac{M}{N}$ can be expressed as $\\left[\\frac{a}{b}, \\frac{c}{d}\\right)$ where $a, b, c, d$ are positive integers with $\\operatorname{gcd}(a, b)=\\operatorname{gcd}(c, d)=1$. Compute $1000 a+100 b+10 c+d$.",
        "answer": "2031",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2276,
        "difficulty": 4.0
      },
      {
        "id": "17c44ea37cdee81bceef8ab9d4a01a96f1c561d0999dc946d6f9a367ad3d0fef",
        "domain": "omni",
        "question": "Let $A B C D$ be a parallelogram with $A B=480, A D=200$, and $B D=625$. The angle bisector of $\\angle B A D$ meets side $C D$ at point $E$. Find $C E$.",
        "answer": "280",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2542,
        "difficulty": 4.0
      },
      {
        "id": "fe46e68a221ec8ff0cdc6a2bbef11d7279997a9bdab57d0e7848e9af65efb6a4",
        "domain": "omni",
        "question": "A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.\n\nInitially, each of the 400 sites is unoccupied. Amy and Ben take turns placing stones with Amy going first. On her turn, Amy places a new red stone on an unoccupied site such that the distance between any two sites occupied by red stones is not equal to $\\sqrt{5}$. On his turn, Ben places a new blue stone on any unoccupied site. (A site occupied by a blue stone is allowed to be at any distance from any other occupied site.) They stop as soon as a player cannot place a stone.\n\nFind the greatest $K$ such that Amy can ensure that she places at least $K$ red stones, no matter how Ben places his blue stones.\n\n[i]",
        "answer": "100",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "imo",
        "source_index": 3830,
        "difficulty": 9.0
      },
      {
        "id": "3f8933d348b37639e09becc8ee9b6f0e2aac515258bc17a2175e106cd5b4411e",
        "domain": "omni",
        "question": "A contest has six problems worth seven points each. On any given problem, a contestant can score either 0,1 , or 7 points. How many possible total scores can a contestant achieve over all six problems?",
        "answer": "28",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1040,
        "difficulty": 5.0
      },
      {
        "id": "e74a5d99b9137c31cf8c15ec3ae24935881cb4cd7e23bc151ed7c69ed2ba3b8c",
        "domain": "omni",
        "question": "Consider all functions $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ satisfying $$f(f(x)+2 x+20)=15$$ Call an integer $n$ good if $f(n)$ can take any integer value. In other words, if we fix $n$, for any integer $m$, there exists a function $f$ such that $f(n)=m$. Find the sum of all good integers $x$.",
        "answer": "-35",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2108,
        "difficulty": 4.0
      },
      {
        "id": "0d58404fa8d95618a7b3957b7524909a304374a69425c7d32bd41edcee330238",
        "domain": "omni",
        "question": "If $x, y$, and $z$ are real numbers such that $2 x^{2}+y^{2}+z^{2}=2 x-4 y+2 x z-5$, find the maximum possible value of $x-y+z$.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 472,
        "difficulty": 5.0
      },
      {
        "id": "3007a711c70625ecaccd37c03b4dc9f21d59e847a49d8995e3ec7546b387e14f",
        "domain": "omni",
        "question": "The line with equation $y=3x+6$ is reflected in the $y$-axis. What is the $x$-intercept of the new line?",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "fermat",
        "source_index": 2797,
        "difficulty": 2.0
      },
      {
        "id": "db67cc6f09425a9aad947762b526711018b3ea5e8c1e0b3ef0cee2aa52249a7d",
        "domain": "omni",
        "question": "What is the smallest possible value of $n$ if a solid cube is made of white plastic and has dimensions $n \\times n \\times n$, the six faces of the cube are completely covered with gold paint, the cube is then cut into $n^{3}$ cubes, each of which has dimensions $1 \\times 1 \\times 1$, and the number of $1 \\times 1 \\times 1$ cubes with 0 gold faces is strictly greater than the number of $1 \\times 1 \\times 1$ cubes with exactly 1 gold face?",
        "answer": "9",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "pascal",
        "source_index": 3023,
        "difficulty": 2.5
      },
      {
        "id": "641e8ec61c6627a7d6f6f2c19d6a650eddc4a24eedad7b2a326dde356ee68170",
        "domain": "omni",
        "question": "While waiting for their food at a restaurant in Harvard Square, Ana and Banana draw 3 squares $\\square_{1}, \\square_{2}, \\square_{3}$ on one of their napkins. Starting with Ana, they take turns filling in the squares with integers from the set $\\{1,2,3,4,5\\}$ such that no integer is used more than once. Ana's goal is to minimize the minimum value $M$ that the polynomial $a_{1} x^{2}+a_{2} x+a_{3}$ attains over all real $x$, where $a_{1}, a_{2}, a_{3}$ are the integers written in $\\square_{1}, \\square_{2}, \\square_{3}$ respectively. Banana aims to maximize $M$. Assuming both play optimally, compute the final value of $100 a_{1}+10 a_{2}+a_{3}$.",
        "answer": "\\[ 541 \\]",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 334,
        "difficulty": 5.0
      },
      {
        "id": "0ed351252290629f240c2b7e3eb53a290554104ad5185af1a7b56e16df969e97",
        "domain": "omni",
        "question": "If $x, y, k$ are positive reals such that $$3=k^{2}\\left(\\frac{x^{2}}{y^{2}}+\\frac{y^{2}}{x^{2}}\\right)+k\\left(\\frac{x}{y}+\\frac{y}{x}\\right)$$ find the maximum possible value of $k$.",
        "answer": "(-1+\\sqrt{7})/2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1339,
        "difficulty": 5.0
      },
      {
        "id": "4509ab8d2f92ee2cb95eb2e64a00224b4dfae015d3e214594d507424db21fef1",
        "domain": "omni",
        "question": "Find the total number of solutions to the equation $(a-b)(a+b)+(a-b)(c)=(a-b)(a+b+c)=2012$ where $a, b, c$ are positive integers.",
        "answer": "1755",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2395,
        "difficulty": 4.0
      },
      {
        "id": "f5f009a813095fb18a227a5e1035301b71cf722f877d4b6e920cdbcb33b2cc0e",
        "domain": "omni",
        "question": "Descartes's Blackjack: How many integer lattice points (points of the form $(m, n)$ for integers $m$ and $n$) lie inside or on the boundary of the disk of radius 2009 centered at the origin?",
        "answer": "12679605",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1537,
        "difficulty": 5.0
      },
      {
        "id": "6fa3096cb814635948cc46f5ee26b00d2f7e4184ae48612b61cdaff73aa6de3a",
        "domain": "omni",
        "question": "Arturo has an equal number of $\\$5$ bills, of $\\$10$ bills, and of $\\$20$ bills. The total value of these bills is $\\$700$. How many $\\$5$ bills does Arturo have?",
        "answer": "20",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 2736,
        "difficulty": 1.5
      },
      {
        "id": "4f8fc97346a6db1131b47789bd1995655783352a99cbeeff6aa3adbd9ff9b76e",
        "domain": "omni",
        "question": "For positive integers $n$, let $c_{n}$ be the smallest positive integer for which $n^{c_{n}}-1$ is divisible by 210, if such a positive integer exists, and $c_{n}=0$ otherwise. What is $c_{1}+c_{2}+\\cdots+c_{210}$?",
        "answer": "329",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_2",
        "source_index": 969,
        "difficulty": 5.0
      },
      {
        "id": "e78618bb328cf1c2ec66ab7472338e2fc6cb9fa7b008297f9f30053e80c4c393",
        "domain": "omni",
        "question": "For the specific example $M=5$, find a value of $k$, not necessarily the smallest, such that $\\sum_{n=1}^{k} \\frac{1}{n}>M$. Justify your answer.",
        "answer": "256",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_11",
        "source_index": 1897,
        "difficulty": 5.0
      },
      {
        "id": "cc6b975017651793505d085a9dbe011061a40a98c7644ed737350c90e2588927",
        "domain": "omni",
        "question": "Define a sequence $\\left\\{a_{n}\\right\\}$ by $a_{1}=1$ and $a_{n}=\\left(a_{n-1}\\right)!+1$ for every $n>1$. Find the least $n$ for which $a_{n}>10^{10}$.",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Sequences and Series -> Other",
        "source": "HMMT_11",
        "source_index": 2547,
        "difficulty": 5.25
      },
      {
        "id": "015c30bfd4af20815ebbf9390b3599f6a93c0c0b88642ac797adf5731b59f56e",
        "domain": "omni",
        "question": "What is the smallest value that the sum of the digits of the number $3n^2+n+1,$ $n\\in\\mathbb{N}$ can take?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "danube_mathematical_olympiad",
        "source_index": 3797,
        "difficulty": 5.5
      },
      {
        "id": "b81271b78851820ab1af97e26d136cb2922f9ecdcac4cfb51e9c10be16070b91",
        "domain": "omni",
        "question": "A positive integer will be called \"sparkly\" if its smallest (positive) divisor, other than 1, equals the total number of divisors (including 1). How many of the numbers $2,3, \\ldots, 2003$ are sparkly?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 491,
        "difficulty": 4.0
      },
      {
        "id": "f1b1ff584548e82031b3279d45dcb7de7e8dba986f5634cd0c4849c32dbaf48d",
        "domain": "omni",
        "question": "Let $a, b, c$ be the three roots of $p(x)=x^{3}+x^{2}-333 x-1001$. Find $a^{3}+b^{3}+c^{3}$.",
        "answer": "2003",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1316,
        "difficulty": 4.5
      },
      {
        "id": "375ac0d728ec392b34f1b2cf4b01d8748153e1aad8f55f91981c9dfef99c0d6e",
        "domain": "omni",
        "question": "What is the area of rectangle \\( PQRS \\) if the perimeter of rectangle \\( TVWY \\) is 60?",
        "answer": "600",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "pascal",
        "source_index": 2925,
        "difficulty": 2.0
      },
      {
        "id": "67f4037ef5efa8595bb85d066e84c9d47692fdc20e0ee35cd240fb5735f06c4a",
        "domain": "omni",
        "question": "In rectangle $PQRS$, $PS=6$ and $SR=3$. Point $U$ is on $QR$ with $QU=2$. Point $T$ is on $PS$ with $\\angle TUR=90^{\\circ}$. What is the length of $TR$?",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "pascal",
        "source_index": 2911,
        "difficulty": 2.0
      },
      {
        "id": "e34fb73787a2e76f344c27c68ff94564aa7acf790da1082ce1223aa916f675f5",
        "domain": "omni",
        "question": "We have a calculator with two buttons that displays an integer $x$. Pressing the first button replaces $x$ by $\\left\\lfloor\\frac{x}{2}\\right\\rfloor$, and pressing the second button replaces $x$ by $4 x+1$. Initially, the calculator displays 0. How many integers less than or equal to 2014 can be achieved through a sequence of arbitrary button presses? (It is permitted for the number displayed to exceed 2014 during the sequence. Here, $\\lfloor y\\rfloor$ denotes the greatest integer less than or equal to the real number $y$.)",
        "answer": "233",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1115,
        "difficulty": 4.5
      },
      {
        "id": "9e4067b070e50caf27c16a54602a2a97c3d69e6565fa4f5441b5de0195bfb06d",
        "domain": "omni",
        "question": "For how many odd integers $k$ between 0 and 100 does the equation $2^{4m^{2}}+2^{m^{2}-n^{2}+4}=2^{k+4}+2^{3m^{2}+n^{2}+k}$ have exactly two pairs of positive integers $(m, n)$ that are solutions?",
        "answer": "18",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 3179,
        "difficulty": 3.0
      },
      {
        "id": "473495adcd56578f1844b57900972591321fd244ca2816330fda0c14834c914f",
        "domain": "omni",
        "question": "Six soccer teams are competing in a tournament in Waterloo. Every team is to play three games, each against a different team. How many different schedules are possible?",
        "answer": "70",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 2801,
        "difficulty": 2.0
      },
      {
        "id": "db75857868a6c1cdf935af40e9358d3a67dc1524354e1af95677b4953e4d642f",
        "domain": "omni",
        "question": "Start by writing the integers $1,2,4,6$ on the blackboard. At each step, write the smallest positive integer $n$ that satisfies both of the following properties on the board. - $n$ is larger than any integer on the board currently. - $n$ cannot be written as the sum of 2 distinct integers on the board. Find the 100-th integer that you write on the board. Recall that at the beginning, there are already 4 integers on the board.",
        "answer": "388",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_2",
        "source_index": 644,
        "difficulty": 5.0
      },
      {
        "id": "2584ce115d066f179012f321a93f1817259b25532d310d47abe5e8350f0d9401",
        "domain": "omni",
        "question": "A $5 \\times 5$ grid of squares is filled with integers. Call a rectangle corner-odd if its sides are grid lines and the sum of the integers in its four corners is an odd number. What is the maximum possible number of corner-odd rectangles within the grid?",
        "answer": "60",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1942,
        "difficulty": 5.0
      },
      {
        "id": "dec13de7f76c0beb267e8ea4346555303ba1f3dc1ae80584f878bf119e5f2f90",
        "domain": "omni",
        "question": "Find all integers $n$ satisfying $n \\geq 2$ and $\\dfrac{\\sigma(n)}{p(n)-1} = n$, in which $\\sigma(n)$ denotes the sum of all positive divisors of $n$, and $p(n)$ denotes the largest prime divisor of $n$.",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "apmo",
        "source_index": 3563,
        "difficulty": 8.0
      },
      {
        "id": "e12bca5335fd7b6584d0d7f258765ff019fef5b2c867c1859c04368b72af7bac",
        "domain": "omni",
        "question": "Consider the paths from \\((0,0)\\) to \\((6,3)\\) that only take steps of unit length up and right. Compute the sum of the areas bounded by the path, the \\(x\\)-axis, and the line \\(x=6\\) over all such paths.",
        "answer": "756",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2310,
        "difficulty": 5.0
      },
      {
        "id": "daeb829b7247c972b8de15530816d3d067d427851aa994b8663c12d63feddf01",
        "domain": "omni",
        "question": "A circle $\\omega_{1}$ of radius 15 intersects a circle $\\omega_{2}$ of radius 13 at points $P$ and $Q$. Point $A$ is on line $P Q$ such that $P$ is between $A$ and $Q$. $R$ and $S$ are the points of tangency from $A$ to $\\omega_{1}$ and $\\omega_{2}$, respectively, such that the line $A S$ does not intersect $\\omega_{1}$ and the line $A R$ does not intersect $\\omega_{2}$. If $P Q=24$ and $\\angle R A S$ has a measure of $90^{\\circ}$, compute the length of $A R$.",
        "answer": "14+\\sqrt{97}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 1695,
        "difficulty": 4.0
      },
      {
        "id": "db6362afb4dac01683d583c84a59bf0c0492967bd0c05663518dcd633422473d",
        "domain": "omni",
        "question": "Let $\\omega$ be a circle, and let $ABCD$ be a quadrilateral inscribed in $\\omega$. Suppose that $BD$ and $AC$ intersect at a point $E$. The tangent to $\\omega$ at $B$ meets line $AC$ at a point $F$, so that $C$ lies between $E$ and $F$. Given that $AE=6, EC=4, BE=2$, and $BF=12$, find $DA$.",
        "answer": "2 \\sqrt{42}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 886,
        "difficulty": 5.25
      },
      {
        "id": "ed3b66a06946aad4699d2b3070a3bc041138f804709199e57a6c87a10164b16e",
        "domain": "omni",
        "question": "Matilda has a summer job delivering newspapers. She earns \\$6.00 an hour plus \\$0.25 per newspaper delivered. Matilda delivers 30 newspapers per hour. How much money will she earn during a 3-hour shift?",
        "answer": "\\$40.50",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3396,
        "difficulty": 1.5
      },
      {
        "id": "ab489bea08504a302df5c1559e2fb85ed5d6753aae3dc31cd4a4be583c3e59db",
        "domain": "omni",
        "question": "The polynomial \\( f(x)=x^{2007}+17 x^{2006}+1 \\) has distinct zeroes \\( r_{1}, \\ldots, r_{2007} \\). A polynomial \\( P \\) of degree 2007 has the property that \\( P\\left(r_{j}+\\frac{1}{r_{j}}\\right)=0 \\) for \\( j=1, \\ldots, 2007 \\). Determine the value of \\( P(1) / P(-1) \\).",
        "answer": "289/259",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1384,
        "difficulty": 6.0
      },
      {
        "id": "6158dd08ce0647fceab4b732e64f0de7cb0ff0fd48186abc7ba48a2a55a02f0c",
        "domain": "omni",
        "question": "For a positive integer $n$, let $\\theta(n)$ denote the number of integers $0 \\leq x<2010$ such that $x^{2}-n$ is divisible by 2010. Determine the remainder when $\\sum_{n=0}^{2009} n \\cdot \\theta(n)$ is divided by 2010.",
        "answer": "335",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 502,
        "difficulty": 5.0
      },
      {
        "id": "986a761c68e06c9a78f9ffbbc9d17fb9aeb36eeaefc7b7f500e3cb3aa17ee4c1",
        "domain": "omni",
        "question": "The digits in a two-digit positive integer are reversed. The new two-digit integer minus the original integer equals 54. What is the positive difference between the two digits of the original integer?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 3054,
        "difficulty": 2.5
      },
      {
        "id": "a929a82c58e80108633ad21a255a28a88c75a7783dc459c265e409af61b0868a",
        "domain": "omni",
        "question": "A ball inside a rectangular container of width 7 and height 12 is launched from the lower-left vertex of the container. It first strikes the right side of the container after traveling a distance of $\\sqrt{53}$ (and strikes no other sides between its launch and its impact with the right side). Find the height at which the ball first contacts the right side.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1770,
        "difficulty": 4.0
      },
      {
        "id": "393e07a9707018a3fe24e719726709070cd321ffe43aed05808bda58ab78e7ec",
        "domain": "omni",
        "question": "Suppose \\(x\\) and \\(y\\) are positive real numbers such that \\(x+\\frac{1}{y}=y+\\frac{2}{x}=3\\). Compute the maximum possible value of \\(xy\\).",
        "answer": "3+\\sqrt{7}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2489,
        "difficulty": 4.5
      },
      {
        "id": "851c2c6a751171e4b4437cae08aeaba9ebe4acb8184ee69e99c82a639dde3f30",
        "domain": "omni",
        "question": "The value of $\\frac{x}{2}$ is less than the value of $x^{2}$. The value of $x^{2}$ is less than the value of $x$. Which of the following could be a value of $x$?",
        "answer": "\\frac{3}{4}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "pascal",
        "source_index": 3504,
        "difficulty": 1.5
      },
      {
        "id": "157ff684c03baf209b29a443d45f2b4407175332231f2cf1d5ab0ebed6c45f46",
        "domain": "omni",
        "question": "Let $W$ be the hypercube $\\left\\{\\left(x_{1}, x_{2}, x_{3}, x_{4}\\right) \\mid 0 \\leq x_{1}, x_{2}, x_{3}, x_{4} \\leq 1\\right\\}$. The intersection of $W$ and a hyperplane parallel to $x_{1}+x_{2}+x_{3}+x_{4}=0$ is a non-degenerate 3-dimensional polyhedron. What is the maximum number of faces of this polyhedron?",
        "answer": "8",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 602,
        "difficulty": 4.5
      },
      {
        "id": "f15545077c34283d5eee8891303d6e4393d4e98d84efa11bcb2b76c9173a40c3",
        "domain": "omni",
        "question": "An ant starts at the origin of a coordinate plane. Each minute, it either walks one unit to the right or one unit up, but it will never move in the same direction more than twice in the row. In how many different ways can it get to the point $(5,5)$ ?",
        "answer": "84",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2652,
        "difficulty": 5.0
      },
      {
        "id": "7032ef7fbc914d83e28d8b7558201266f5a3dd64c6fea077c0c9ef3788879e59",
        "domain": "omni",
        "question": "Let $G$ be the number of Google hits of \"guts round\" at 10:31PM on October 31, 2011. Let $B$ be the number of Bing hits of \"guts round\" at the same time. Determine $B / G$. Your score will be $$\\max (0,\\left\\lfloor 20\\left(1-\\frac{20|a-k|}{k}\\right)\\right\\rfloor)$$ where $k$ is the actual answer and $a$ is your answer.",
        "answer": ".82721",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2048,
        "difficulty": 3.5
      },
      {
        "id": "a3306fd8cb5be7bca8d17ef870dc343c018375a9c1129272614d55eaee8b4392",
        "domain": "omni",
        "question": "Let $x$ and $y$ be non-negative real numbers that sum to 1. Compute the number of ordered pairs $(a, b)$ with $a, b \\in\\{0,1,2,3,4\\}$ such that the expression $x^{a} y^{b}+y^{a} x^{b}$ has maximum value $2^{1-a-b}$.",
        "answer": "17",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2036,
        "difficulty": 5.0
      },
      {
        "id": "792a4121296dab026fe5a03f6fba3fb6997e56ebb203d9952c7f28c4c129bf3d",
        "domain": "omni",
        "question": "Let $S(x)$ denote the sum of the digits of a positive integer $x$. Find the maximum possible value of $S(x+2019)-S(x)$.",
        "answer": "12",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Prealgebra -> Other",
        "source": "HMMT_2",
        "source_index": 928,
        "difficulty": 5.25
      },
      {
        "id": "1c2b67955382a9175fff4c416ca77e0fbdbd27d3b149f877ce690e6f667aa749",
        "domain": "omni",
        "question": "A particular coin can land on heads $(H)$, on tails $(T)$, or in the middle $(M)$, each with probability $\\frac{1}{3}$. Find the expected number of flips necessary to observe the contiguous sequence HMMTHMMT...HMMT, where the sequence HMMT is repeated 2016 times.",
        "answer": "\\frac{3^{8068}-81}{80}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1026,
        "difficulty": 5.25
      },
      {
        "id": "b4e752626f21bd735954ffcdc599e77ce98e25ae3ba5efaaaa65bf95eea900ee",
        "domain": "omni",
        "question": "A line with a slope of 2 and a line with a slope of -4 each have a $y$-intercept of 6. What is the distance between the $x$-intercepts of these lines?",
        "answer": "\\frac{9}{2}",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Determinants",
        "source": "fermat",
        "source_index": 2824,
        "difficulty": 2.5
      },
      {
        "id": "00ce959a6819027353608d9ba39ddb0078c867760b6a7c1d7449aebb4084be24",
        "domain": "omni",
        "question": "Herbert rolls 6 fair standard dice and computes the product of all of his rolls. If the probability that the product is prime can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
        "answer": "2692",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1313,
        "difficulty": 4.5
      },
      {
        "id": "34ba729f0c8cad945d3430727fcd35ef0c53ef91f6d873bb893ee9056353d20d",
        "domain": "omni",
        "question": "An ordered list of four numbers is called a quadruple. A quadruple $(p, q, r, s)$ of integers with $p, q, r, s \\geq 0$ is chosen at random such that $2 p+q+r+s=4$. What is the probability that $p+q+r+s=3$?",
        "answer": "\\frac{3}{11}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2861,
        "difficulty": 2.0
      },
      {
        "id": "e728b3c766a3560d7d8ca62ed4b6e8e4304197b53c933bd389a2c38c14672093",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=3, B C=4$, and $C A=5$. Let $A_{1}, A_{2}$ be points on side $B C$, $B_{1}, B_{2}$ be points on side $C A$, and $C_{1}, C_{2}$ be points on side $A B$. Suppose that there exists a point $P$ such that $P A_{1} A_{2}, P B_{1} B_{2}$, and $P C_{1} C_{2}$ are congruent equilateral triangles. Find the area of convex hexagon $A_{1} A_{2} B_{1} B_{2} C_{1} C_{2}$.",
        "answer": "\\frac{12+22 \\sqrt{3}}{15}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 855,
        "difficulty": 5.25
      },
      {
        "id": "3b9c67c08dbe56b3c066d44b2253068cf379782bf81467c02ee77236701d135c",
        "domain": "omni",
        "question": "A polynomial $P$ of degree 2015 satisfies the equation $P(n)=\\frac{1}{n^{2}}$ for $n=1,2, \\ldots, 2016$. Find \\lfloor 2017 P(2017)\\rfloor.",
        "answer": "-9",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1649,
        "difficulty": 6.0
      },
      {
        "id": "7f595c3dcdef2c0e02831f8dbb1494b05c87e51421e4748498513c0653af4c86",
        "domain": "omni",
        "question": "In a photograph, Aristotle, David, Flora, Munirah, and Pedro are seated in a random order in a row of 5 chairs. If David is seated in the middle of the row, what is the probability that Pedro is seated beside him?",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "fermat",
        "source_index": 3088,
        "difficulty": 2.5
      },
      {
        "id": "7cafc391b54397271e1cc629e59934c6ae13c21b42f087e9a9cb566511f424ab",
        "domain": "omni",
        "question": "Let \\{a_{n}\\}_{n \\geq 1}$ be an arithmetic sequence and \\{g_{n}\\}_{n \\geq 1}$ be a geometric sequence such that the first four terms of \\{a_{n}+g_{n}\\}$ are $0,0,1$, and 0 , in that order. What is the 10th term of \\{a_{n}+g_{n}\\}$ ?",
        "answer": "-54",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series",
        "source": "HMMT_2",
        "source_index": 1586,
        "difficulty": 5.0
      },
      {
        "id": "dc7f2bc69b2718b08ae94c8e3cd7052d8a64fda7cee04f945b419da21016f224",
        "domain": "omni",
        "question": "The real numbers $x, y, z$ satisfy $0 \\leq x \\leq y \\leq z \\leq 4$. If their squares form an arithmetic progression with common difference 2, determine the minimum possible value of $|x-y|+|y-z|$.",
        "answer": "4-2\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "HMMT_2",
        "source_index": 637,
        "difficulty": 3.5
      },
      {
        "id": "1c17298e7847abb0e6627a66e7a00510d7d210fad12d6edd481fe37039005f5e",
        "domain": "omni",
        "question": "In preparation for a game of Fish, Carl must deal 48 cards to 6 players. For each card that he deals, he runs through the entirety of the following process: 1. He gives a card to a random player. 2. A player Z is randomly chosen from the set of players who have at least as many cards as every other player (i.e. Z has the most cards or is tied for having the most cards). 3. A player D is randomly chosen from the set of players other than Z who have at most as many cards as every other player (i.e. D has the fewest cards or is tied for having the fewest cards). 4. Z gives one card to D. He repeats steps 1-4 for each card dealt, including the last card. After all the cards have been dealt, what is the probability that each player has exactly 8 cards?",
        "answer": "\\frac{5}{6}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2112,
        "difficulty": 4.0
      },
      {
        "id": "0c5021bdb2fa331a97227a28736cfe0241224f3c7dd3b42f3be4a956c8514a47",
        "domain": "omni",
        "question": "A wealthy king has his blacksmith fashion him a large cup, whose inside is a cone of height 9 inches and base diameter 6 inches. At one of his many feasts, he orders the mug to be filled to the brim with cranberry juice. For each positive integer $n$, the king stirs his drink vigorously and takes a sip such that the height of fluid left in his cup after the sip goes down by $\\frac{1}{n^{2}}$ inches. Shortly afterwards, while the king is distracted, the court jester adds pure Soylent to the cup until it's once again full. The king takes sips precisely every minute, and his first sip is exactly one minute after the feast begins. As time progresses, the amount of juice consumed by the king (in cubic inches) approaches a number $r$. Find $r$.",
        "answer": "\\frac{216 \\pi^{3}-2187 \\sqrt{3}}{8 \\pi^{2}}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Multi-variable; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 683,
        "difficulty": 5.0
      },
      {
        "id": "90229e286b3d1618f72d8af819b0e21fc1067fae4dab7f8363dd915feb1f921a",
        "domain": "omni",
        "question": "In triangle $A B C$ with altitude $A D, \\angle B A C=45^{\\circ}, D B=3$, and $C D=2$. Find the area of triangle $A B C$.",
        "answer": "15",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1189,
        "difficulty": 5.0
      },
      {
        "id": "7d72371fd9809ca7339b38fdce989a2e88e3b14df51f1b4f9f6199535c816184",
        "domain": "omni",
        "question": "There are 2008 distinct points on a circle. If you connect two of these points to form a line and then connect another two points (distinct from the first two) to form another line, what is the probability that the two lines intersect inside the circle?",
        "answer": "1/3",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1466,
        "difficulty": 4.5
      },
      {
        "id": "ac54c3533c035204c0d87787d27101b5610dcec630878e8f09d111da41d46390",
        "domain": "omni",
        "question": "Simplify $\frac{1}{2+\frac{2}{3}}$.",
        "answer": "\\frac{3}{8}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3398,
        "difficulty": 1.5
      },
      {
        "id": "5b0b5d2d9290ae5188c4b6e3979e811e3e5b1b2e1941caf1a01a6e4161f8de27",
        "domain": "omni",
        "question": "Let $A=\\{a_{1}, b_{1}, a_{2}, b_{2}, \\ldots, a_{10}, b_{10}\\}$, and consider the 2-configuration $C$ consisting of \\( \\{a_{i}, b_{i}\\} \\) for all \\( 1 \\leq i \\leq 10, \\{a_{i}, a_{i+1}\\} \\) for all \\( 1 \\leq i \\leq 9 \\), and \\( \\{b_{i}, b_{i+1}\\} \\) for all \\( 1 \\leq i \\leq 9 \\). Find the number of subsets of $C$ that are consistent of order 1.",
        "answer": "89",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1048,
        "difficulty": 4.5
      },
      {
        "id": "96be2167ec604e69544c0b2792b83391ccfd2f12e56a2a7d9f171630c68e0797",
        "domain": "omni",
        "question": "In some squares of a $2012\\times 2012$ grid there are some beetles, such that no square contain more than one beetle. At one moment, all the beetles fly off the grid and then land on the grid again, also satisfying the condition that there is at most one beetle standing in each square. The vector from the centre of the square from which a beetle $B$ flies to the centre of the square on which it lands is called the [i]translation vector[/i] of beetle $B$. \nFor all possible starting and ending configurations, find the maximum length of the sum of the  [i]translation vectors[/i] of all beetles.",
        "answer": "\\frac{2012^3}{4}",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Vectors; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_team_selection_test",
        "source_index": 522,
        "difficulty": 8.0
      },
      {
        "id": "1ea182428fe205f06653e63e3ba9edfb6aecc87c0ad4867ae4ca014e927d9500",
        "domain": "omni",
        "question": "In \\(\\triangle ABC\\), the external angle bisector of \\(\\angle BAC\\) intersects line \\(BC\\) at \\(D\\). \\(E\\) is a point on ray \\(\\overrightarrow{AC}\\) such that \\(\\angle BDE=2 \\angle ADB\\). If \\(AB=10, AC=12\\), and \\(CE=33\\), compute \\(\\frac{DB}{DE}\\).",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2187,
        "difficulty": 5.0
      },
      {
        "id": "1d3ec0071e0cf4d56c426fa904fc188f8f9c45ccdb8daebb1d00a181ea005a22",
        "domain": "omni",
        "question": "Let $x, y, z$ be real numbers satisfying $$\\begin{aligned} 2 x+y+4 x y+6 x z & =-6 \\\\ y+2 z+2 x y+6 y z & =4 \\\\ x-z+2 x z-4 y z & =-3 \\end{aligned}$$ Find $x^{2}+y^{2}+z^{2}$.",
        "answer": "29",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2589,
        "difficulty": 4.75
      },
      {
        "id": "be30143c30f1ab514853738753f1f116270e8acdadfa9cddc308484dfe05a652",
        "domain": "omni",
        "question": "A group of 101 Dalmathians participate in an election, where they each vote independently on either candidate \\(A\\) or \\(B\\) with equal probability. If \\(X\\) Dalmathians voted for the winning candidate, the expected value of \\(X^{2}\\) can be expressed as \\(\\frac{a}{b}\\) for positive integers \\(a, b\\) with \\(\\operatorname{gcd}(a, b)=1\\). Find the unique positive integer \\(k \\leq 103\\) such that \\(103 \\mid a-bk\\).",
        "answer": "51",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2461,
        "difficulty": 5.25
      },
      {
        "id": "84180647b8ca612979ce0971a805c04f35f0028b595359ae20ad4aec9382f6ee",
        "domain": "omni",
        "question": "Consider an equilateral triangular grid $G$ with 20 points on a side, where each row consists of points spaced 1 unit apart. More specifically, there is a single point in the first row, two points in the second row, ..., and 20 points in the last row, for a total of 210 points. Let $S$ be a closed non-selfintersecting polygon which has 210 vertices, using each point in $G$ exactly once. Find the sum of all possible values of the area of $S$.",
        "answer": "52 \\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 555,
        "difficulty": 5.25
      },
      {
        "id": "d302b8d05ce38ae4d538369dd10c53f9e208e6c64bd10f28937ee6251c3c582c",
        "domain": "omni",
        "question": "How many ways are there to label the faces of a regular octahedron with the integers 18, using each exactly once, so that any two faces that share an edge have numbers that are relatively prime? Physically realizable rotations are considered indistinguishable, but physically unrealizable reflections are considered different.",
        "answer": "12",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations",
        "source": "HMMT_2",
        "source_index": 539,
        "difficulty": 5.25
      },
      {
        "id": "b4123d12fbd516f2f9fae72e370c376266f40f7b68aaeec4da9294a6b4a344b9",
        "domain": "omni",
        "question": "Alex picks his favorite point $(x, y)$ in the first quadrant on the unit circle $x^{2}+y^{2}=1$, such that a ray from the origin through $(x, y)$ is $\\theta$ radians counterclockwise from the positive $x$-axis. He then computes $\\cos ^{-1}\\left(\\frac{4 x+3 y}{5}\\right)$ and is surprised to get $\\theta$. What is $\\tan (\\theta)$?",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1006,
        "difficulty": 6.0
      },
      {
        "id": "fb6451c4fa02cd8249ba08e6c12ff33a94fc6c932c9277dbe8697aa89a323036",
        "domain": "omni",
        "question": "Dorothea has a $3 \\times 4$ grid of dots. She colors each dot red, blue, or dark gray. Compute the number of ways Dorothea can color the grid such that there is no rectangle whose sides are parallel to the grid lines and whose vertices all have the same color.",
        "answer": "284688",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1896,
        "difficulty": 4.5
      },
      {
        "id": "a8c1d31ebd4e8e799f8ff60e3b0c5bb91460fad17f684f9271cc3d10c602142a",
        "domain": "omni",
        "question": "Let $a_{0}, a_{1}, \\ldots$ and $b_{0}, b_{1}, \\ldots$ be geometric sequences with common ratios $r_{a}$ and $r_{b}$, respectively, such that $$\\sum_{i=0}^{\\infty} a_{i}=\\sum_{i=0}^{\\infty} b_{i}=1 \\quad \\text { and } \\quad\\left(\\sum_{i=0}^{\\infty} a_{i}^{2}\\right)\\left(\\sum_{i=0}^{\\infty} b_{i}^{2}\\right)=\\sum_{i=0}^{\\infty} a_{i} b_{i}$$ Find the smallest real number $c$ such that $a_{0}<c$ must be true.",
        "answer": "\\frac{4}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals",
        "source": "HMMT_11",
        "source_index": 1966,
        "difficulty": 4.0
      },
      {
        "id": "d78a63f88e456aa14a0d9b710852078a157d1354bed260f4f1c375f66e09f95c",
        "domain": "omni",
        "question": "An icosidodecahedron is a convex polyhedron with 20 triangular faces and 12 pentagonal faces. How many vertices does it have?",
        "answer": "30",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes; Mathematics -> Discrete Mathematics -> Other",
        "source": "HMMT_2",
        "source_index": 1471,
        "difficulty": 4.0
      },
      {
        "id": "fddc7b498e97429ce51ec804c8f04840040ef019d1865ce1bd0653e381fa6630",
        "domain": "omni",
        "question": "Horizontal parallel segments $A B=10$ and $C D=15$ are the bases of trapezoid $A B C D$. Circle $\\gamma$ of radius 6 has center within the trapezoid and is tangent to sides $A B, B C$, and $D A$. If side $C D$ cuts out an arc of $\\gamma$ measuring $120^{\\circ}$, find the area of $A B C D$.",
        "answer": "\\frac{225}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2199,
        "difficulty": 4.0
      },
      {
        "id": "96b4ce0ded4eeac72fc40951f16602dc1ee53cdf2d142811ed8774e1e7045e95",
        "domain": "omni",
        "question": "Let $n$ be a positive integer. Given that $n^{n}$ has 861 positive divisors, find $n$.",
        "answer": "20",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2501,
        "difficulty": 4.0
      },
      {
        "id": "e6b5df0e235208149649d7f167808616e7cf6aae05b4758951d4c067b8607fff",
        "domain": "omni",
        "question": "Two fair coins are simultaneously flipped. This is done repeatedly until at least one of the coins comes up heads, at which point the process stops. What is the probability that the other coin also came up heads on this last flip?",
        "answer": "1/3",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 910,
        "difficulty": 4.0
      },
      {
        "id": "953999a43f6811af6cafca6410c51380e02e5aa404379d934f3af344c3a99f77",
        "domain": "omni",
        "question": "Rahul has ten cards face-down, which consist of five distinct pairs of matching cards. During each move of his game, Rahul chooses one card to turn face-up, looks at it, and then chooses another to turn face-up and looks at it. If the two face-up cards match, the game ends. If not, Rahul flips both cards face-down and keeps repeating this process. Initially, Rahul doesn't know which cards are which. Assuming that he has perfect memory, find the smallest number of moves after which he can guarantee that the game has ended.",
        "answer": "4",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 588,
        "difficulty": 4.0
      },
      {
        "id": "b52754ac6ee26e9d5ec15e6599a1d792f00529e774a16c8d38a71d4eaaf4aebe",
        "domain": "omni",
        "question": "If a line segment joins the points $(-9,-2)$ and $(6,8)$, how many points on the line segment have coordinates that are both integers?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2680,
        "difficulty": 2.0
      },
      {
        "id": "1083c71b1202f4ff6e8ee9d46b30076d81742dd30e9b03a856f7379f2fd4fc07",
        "domain": "omni",
        "question": "Suppose that $a,b,c,d$ are positive real numbers satisfying $(a+c)(b+d)=ac+bd$. Find the smallest possible value of\n$$\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}.$$\n[i]Israel[/i]",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "imo_shortlist",
        "source_index": 3879,
        "difficulty": 7.0
      },
      {
        "id": "6bda79342d059d365d43b0e7813ba79563be72e7fcd5ff66bcbb5a09289f7dcf",
        "domain": "omni",
        "question": "A jar contains 8 red balls and 2 blue balls. Every minute, a ball is randomly removed. The probability that there exists a time during this process where there are more blue balls than red balls in the jar can be expressed as $\\frac{a}{b}$ for relatively prime integers $a$ and $b$. Compute $100 a+b$.",
        "answer": "209",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1357,
        "difficulty": 4.375
      },
      {
        "id": "65621c05c4c6016f7fd739144e505bec8ebac1a99cad658e6c2b8383bfb8b7c6",
        "domain": "omni",
        "question": "Find the smallest $n$ such that $n$! ends in 290 zeroes.",
        "answer": "1170",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1281,
        "difficulty": 4.5
      },
      {
        "id": "a9bd56a923e068ac8ddefa59b7810eb0568afc91b9efd8eaec2b7d4a20ec1e41",
        "domain": "omni",
        "question": "There are four people in a room. For every two people, there is a $50 \\%$ chance that they are friends. Two people are connected if they are friends, or a third person is friends with both of them, or they have different friends who are friends of each other. What is the probability that every pair of people in this room is connected?",
        "answer": "\\frac{19}{32}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "pascal",
        "source_index": 2987,
        "difficulty": 2.0
      },
      {
        "id": "60cae469d4962082015a41891cc1c3310baec1cb3aa47666c8ccb74187a0facd",
        "domain": "omni",
        "question": "Compute \\(\\sum_{k=0}^{100}\\left\\lfloor\\frac{2^{100}}{2^{50}+2^{k}}\\right\\rfloor\\). (Here, if \\(x\\) is a real number, then \\(\\lfloor x\\rfloor\\) denotes the largest integer less than or equal to \\(x\\).)",
        "answer": "101 \\cdot 2^{49}-50",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 892,
        "difficulty": 5.0
      },
      {
        "id": "08aa25d7c28e3ed32bc9811a98b61415554ad48dacc711da1f04f3f3fed14e5b",
        "domain": "omni",
        "question": "Alice and Bob take turns removing balls from a bag containing 10 black balls and 10 white balls, with Alice going first. Alice always removes a black ball if there is one, while Bob removes one of the remaining balls uniformly at random. Once all balls have been removed, the expected number of black balls which Bob has can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "4519",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2041,
        "difficulty": 4.5
      },
      {
        "id": "69d6e27e50dfa20443eb1f73ea49b650c705bd971e135d16af8fe65376341d31",
        "domain": "omni",
        "question": "The time on a cell phone is $3:52$. How many minutes will pass before the phone next shows a time using each of the digits 2, 3, and 5 exactly once?",
        "answer": "91",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3037,
        "difficulty": 1.5
      },
      {
        "id": "b08240559d7f1942e9b4c00679dfe191e131dda34e14b5c6b32054db3f8b0f53",
        "domain": "omni",
        "question": "A sequence of 11 positive real numbers, $a_{1}, a_{2}, a_{3}, \\ldots, a_{11}$, satisfies $a_{1}=4$ and $a_{11}=1024$ and $a_{n}+a_{n-1}=\\frac{5}{2} \\sqrt{a_{n} \\cdot a_{n-1}}$ for every integer $n$ with $2 \\leq n \\leq 11$. For example when $n=7, a_{7}+a_{6}=\\frac{5}{2} \\sqrt{a_{7} \\cdot a_{6}}$. There are $S$ such sequences. What are the rightmost two digits of $S$?",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2804,
        "difficulty": 2.5
      },
      {
        "id": "fed3a163b5c076a5977e3196a8c680f11f28c698fe0087511c99aa6552adc85f",
        "domain": "omni",
        "question": "If a bag contains only green, yellow, and red marbles in the ratio $3: 4: 2$ and 63 of the marbles are not red, how many red marbles are in the bag?",
        "answer": "18",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3100,
        "difficulty": 1.5
      },
      {
        "id": "737178e292ec0c4ab765d8466d755315097df59e5462ecb7848e9e2b68806ef4",
        "domain": "omni",
        "question": "When 100 is divided by a positive integer $x$, the remainder is 10. When 1000 is divided by $x$, what is the remainder?",
        "answer": "10",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3197,
        "difficulty": 2.0
      },
      {
        "id": "9c992b19942eb4388034db8b3b163679cc0fa136487b98c37a8d61607635180a",
        "domain": "omni",
        "question": "Let $A B C$ be an acute scalene triangle with circumcenter $O$ and centroid $G$. Given that $A G O$ is a right triangle, $A O=9$, and $B C=15$, let $S$ be the sum of all possible values for the area of triangle $A G O$. Compute $S^{2}$.",
        "answer": "288",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_2",
        "source_index": 1338,
        "difficulty": 5.25
      },
      {
        "id": "248b5680514676c09f2bcaff946b21abba09fdf2328cec2e850d53f6a772d1fa",
        "domain": "omni",
        "question": "Let $S$ be the set of points $(a, b)$ with $0 \\leq a, b \\leq 1$ such that the equation $x^{4}+a x^{3}-b x^{2}+a x+1=0$ has at least one real root. Determine the area of the graph of $S$.",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1289,
        "difficulty": 4.5
      },
      {
        "id": "8414c257dafe6922e321dfe505671ab1926f356e44c03a674bd6d8a79fce56d5",
        "domain": "omni",
        "question": "What is the tens digit of the smallest positive integer that is divisible by each of 20, 16, and 2016?",
        "answer": "8",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "cayley",
        "source_index": 3254,
        "difficulty": 2.25
      },
      {
        "id": "b23560316cdb384205fcfb68ac7205d3bab57e75e546f0a454ed4c6ca177bdea",
        "domain": "omni",
        "question": "What is the median of the numbers in the list $19^{20}, \\frac{20}{19}, 20^{19}, 2019, 20 \\times 19$?",
        "answer": "2019",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2945,
        "difficulty": 1.5
      },
      {
        "id": "4624f80fcf811adc07c654f3333c7f2457043f1a0597184ae725d25bbc6566b9",
        "domain": "omni",
        "question": "If \\( 10^{x} \\cdot 10^{5}=100^{4} \\), what is the value of \\( x \\)?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "fermat",
        "source_index": 2817,
        "difficulty": 2.5
      },
      {
        "id": "250cb927e95a3fe48d0ea9da9d383a38161a8ef9eef40405dbd81a054976d1c9",
        "domain": "omni",
        "question": "The numbers $2^{0}, 2^{1}, \\cdots, 2^{15}, 2^{16}=65536$ are written on a blackboard. You repeatedly take two numbers on the blackboard, subtract one from the other, erase them both, and write the result of the subtraction on the blackboard. What is the largest possible number that can remain on the blackboard when there is only one number left?",
        "answer": "131069",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 874,
        "difficulty": 5.0
      },
      {
        "id": "77515a9c1378a11685449ef894ce3fda759f47acc8d82f30f015d4cf164d065a",
        "domain": "omni",
        "question": "We want to design a new chess piece, the American, with the property that (i) the American can never attack itself, and (ii) if an American $A_{1}$ attacks another American $A_{2}$, then $A_{2}$ also attacks $A_{1}$. Let $m$ be the number of squares that an American attacks when placed in the top left corner of an 8 by 8 chessboard. Let $n$ be the maximal number of Americans that can be placed on the 8 by 8 chessboard such that no Americans attack each other, if one American must be in the top left corner. Find the largest possible value of $m n$.",
        "answer": "1024",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1895,
        "difficulty": 5.25
      },
      {
        "id": "3f63d5837058bc8197f819387b7b84c020e35784eb045eac746ab7f3cee1c6b8",
        "domain": "omni",
        "question": "The real function $f$ has the property that, whenever $a, b, n$ are positive integers such that $a+b=2^{n}$, the equation $f(a)+f(b)=n^{2}$ holds. What is $f(2002)$?",
        "answer": "96",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 809,
        "difficulty": 4.0
      },
      {
        "id": "f98e105a1df5cbcb052ad5599bbaabb931f4febb57b75ec032db3cd367c8c399",
        "domain": "omni",
        "question": "Anders is solving a math problem, and he encounters the expression $\\sqrt{15!}$. He attempts to simplify this radical by expressing it as $a \\sqrt{b}$ where $a$ and $b$ are positive integers. The sum of all possible distinct values of $ab$ can be expressed in the form $q \\cdot 15!$ for some rational number $q$. Find $q$.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2201,
        "difficulty": 4.0
      },
      {
        "id": "0ff18da849f05fc58ee6f55dc6580ab02320de6415f709254d531d53945632a8",
        "domain": "omni",
        "question": "Let $a_0,a_1,a_2,\\dots $ be a sequence of real numbers such that $a_0=0, a_1=1,$ and for every $n\\geq 2$ there exists $1 \\leq k \\leq n$ satisfying \\[ a_n=\\frac{a_{n-1}+\\dots + a_{n-k}}{k}. \\]Find the maximum possible value of $a_{2018}-a_{2017}$.",
        "answer": "\\frac{2016}{2017^2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 4148,
        "difficulty": 5.5
      },
      {
        "id": "1a5c4187069ac8e60cc3a4fa964e76398a78bf5d862d484ca03ca656366b8eac",
        "domain": "omni",
        "question": "The list $p, q, r, s$ consists of four consecutive integers listed in increasing order. If $p + s = 109$, what is the value of $q + r$?",
        "answer": "109",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2957,
        "difficulty": 1.0
      },
      {
        "id": "c2efd1c4f27bc2cd82b1dc96aa1575331ce8a2dda8ef302f5c2f1b05ee087f83",
        "domain": "omni",
        "question": "If $2x-3=10$, what is the value of $4x$?",
        "answer": "26",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3003,
        "difficulty": 1.0
      },
      {
        "id": "c69be95584c69ba843e7e3a73dc2beb7403f6bc92d8effe76d4b24fd182d4fd8",
        "domain": "omni",
        "question": "Svitlana writes the number 147 on a blackboard. Then, at any point, if the number on the blackboard is $n$, she can perform one of the following three operations: - if $n$ is even, she can replace $n$ with $\\frac{n}{2}$; - if $n$ is odd, she can replace $n$ with $\\frac{n+255}{2}$; and - if $n \\geq 64$, she can replace $n$ with $n-64$. Compute the number of possible values that Svitlana can obtain by doing zero or more operations.",
        "answer": "163",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1645,
        "difficulty": 4.0
      },
      {
        "id": "6b36f8c9aeb8b328d8d2568ea2045ecf4b6af8b1adb15ed195868ff0ce577d14",
        "domain": "omni",
        "question": "Let $\\mathbb{Q}$ be the set of rational numbers. A function $f: \\mathbb{Q} \\to \\mathbb{Q}$ is called aquaesulian if the following property holds: for every $x,y \\in \\mathbb{Q}$,\n\\[ f(x+f(y)) = f(x) + y \\quad \\text{or} \\quad f(f(x)+y) = x + f(y). \\]\nShow that there exists an integer $c$ such that for any aquaesulian function $f$ there are at most $c$ different rational numbers of the form $f(r) + f(-r)$ for some rational number $r$, and find the smallest possible value of $c$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "imo",
        "source_index": 4237,
        "difficulty": 8.0
      },
      {
        "id": "dc0668b77a01a56cc86394a8fd9907a2ae505429660e75d9a571d83a17008dfe",
        "domain": "omni",
        "question": "There are $N$ lockers, labeled from 1 to $N$, placed in clockwise order around a circular hallway. Initially, all lockers are open. Ansoon starts at the first locker and always moves clockwise. When she is at locker $n$ and there are more than $n$ open lockers, she keeps locker $n$ open and closes the next $n$ open lockers, then repeats the process with the next open locker. If she is at locker $n$ and there are at most $n$ lockers still open, she keeps locker $n$ open and closes all other lockers. She continues this process until only one locker is left open. What is the smallest integer $N>2021$ such that the last open locker is locker 1?",
        "answer": "2046",
        "topic": "Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_11",
        "source_index": 2513,
        "difficulty": 5.0
      },
      {
        "id": "4e18d4cc40d43d27fb00b764c222f3f57ea2689a8168b8becaddb694b15128ce",
        "domain": "omni",
        "question": "When three positive integers are added in pairs, the resulting sums are 998, 1050, and 1234. What is the difference between the largest and smallest of the three original positive integers?",
        "answer": "236",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3096,
        "difficulty": 1.5
      },
      {
        "id": "96e89337413bd3a7c226d074b7850ea4072731a64c86f2d5acbd5fe24680b79b",
        "domain": "omni",
        "question": "Points $G$ and $N$ are chosen on the interiors of sides $E D$ and $D O$ of unit square $D O M E$, so that pentagon GNOME has only two distinct side lengths. The sum of all possible areas of quadrilateral $N O M E$ can be expressed as $\\frac{a-b \\sqrt{c}}{d}$, where $a, b, c, d$ are positive integers such that $\\operatorname{gcd}(a, b, d)=1$ and $c$ is square-free (i.e. no perfect square greater than 1 divides $c$ ). Compute $1000 a+100 b+10 c+d$.",
        "answer": "10324",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2287,
        "difficulty": 4.5
      },
      {
        "id": "5f1f6b662baa2ec0c3ee3c26303bec98aa0722499e6ae41daa900b976e03d630",
        "domain": "omni",
        "question": "The roots of $z^{6}+z^{4}+z^{2}+1=0$ are the vertices of a convex polygon in the complex plane. Find the sum of the squares of the side lengths of the polygon.",
        "answer": "12-4 \\sqrt{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1481,
        "difficulty": 4.5
      },
      {
        "id": "bb52b45d35d587b0ea323b011113b72f086252cb446b9309772f777f4b32a9c5",
        "domain": "omni",
        "question": "If $a, b, c>0$, what is the smallest possible value of $\\left\\lfloor\\frac{a+b}{c}\\right\\rfloor+\\left\\lfloor\\frac{b+c}{a}\\right\\rfloor+\\left\\lfloor\\frac{c+a}{b}\\right\\rfloor$? (Note that $\\lfloor x\\rfloor$ denotes the greatest integer less than or equal to $x$.)",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "HMMT_2",
        "source_index": 1264,
        "difficulty": 5.0
      },
      {
        "id": "1e379f9844622924a5b497dd88cb31322a5d1435b71e9d004011048231bf8ec1",
        "domain": "omni",
        "question": "A real number $x$ is chosen uniformly at random from the interval $(0,10)$. Compute the probability that $\\sqrt{x}, \\sqrt{x+7}$, and $\\sqrt{10-x}$ are the side lengths of a non-degenerate triangle.",
        "answer": "\\frac{22}{25}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1783,
        "difficulty": 5.0
      },
      {
        "id": "564dfa1539e2565a3e19fbadf1f2b15de8755e9cbfa51af3843f67a99b6f0812",
        "domain": "omni",
        "question": "Let $x$ and $y$ be positive real numbers such that $x^{2}+y^{2}=1$ and \\left(3 x-4 x^{3}\\right)\\left(3 y-4 y^{3}\\right)=-\\frac{1}{2}$. Compute $x+y$.",
        "answer": "\\frac{\\sqrt{6}}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 904,
        "difficulty": 4.0
      },
      {
        "id": "e0a149055ff41a9a97a88302b65258ee30f844343f23c69e408c148343e26899",
        "domain": "omni",
        "question": "Mike has two containers. One container is a rectangular prism with width 2 cm, length 4 cm, and height 10 cm. The other is a right cylinder with radius 1 cm and height 10 cm. Both containers sit on a flat surface. Water has been poured into the two containers so that the height of the water in both containers is the same. If the combined volume of the water in the two containers is $80 \\mathrm{~cm}^{3}$, what is the height of the water in each container?",
        "answer": "7.2",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume",
        "source": "pascal",
        "source_index": 2910,
        "difficulty": 2.0
      },
      {
        "id": "eb40fb52e1f7ff20f2ca55deb7f1c749a90a6fc4498e4fb0e7376b2a4a7e3f40",
        "domain": "omni",
        "question": "If $3^{2x}=64$, what is the value of $3^{-x}$?",
        "answer": "\\frac{1}{8}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "fermat",
        "source_index": 2781,
        "difficulty": 2.5
      },
      {
        "id": "81774616a11a1a90202dcddb1114387b68705e461d89976e67bd7c55a3a98d65",
        "domain": "omni",
        "question": "Solve in positive real numbers: $n+ \\lfloor \\sqrt{n} \\rfloor+\\lfloor \\sqrt[3]{n} \\rfloor=2014$",
        "answer": "1958",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Number Theory -> Other",
        "source": "jbmo_shortlist",
        "source_index": 4412,
        "difficulty": 5.5
      },
      {
        "id": "dd3dac7bf6608096502c7ca24a408840a9dfb89bc47e1d6c0a32da7a196217a2",
        "domain": "omni",
        "question": "In $\\triangle Q R S$, point $T$ is on $Q S$ with $\\angle Q R T=\\angle S R T$. Suppose that $Q T=m$ and $T S=n$ for some integers $m$ and $n$ with $n>m$ and for which $n+m$ is a multiple of $n-m$. Suppose also that the perimeter of $\\triangle Q R S$ is $p$ and that the number of possible integer values for $p$ is $m^{2}+2 m-1$. What is the value of $n-m$?",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "fermat",
        "source_index": 2794,
        "difficulty": 3.0
      },
      {
        "id": "0ade695bd83c2e856a1d72e243bd2cad1736c1e048704aa4aa00461a2005c4c1",
        "domain": "omni",
        "question": "Let $a_{1}=1$, and let $a_{n}=\\left\\lfloor n^{3} / a_{n-1}\\right\\rfloor$ for $n>1$. Determine the value of $a_{999}$.",
        "answer": "999",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1500,
        "difficulty": 5.0
      },
      {
        "id": "77f4ce6d7abdec0027e1d8925edef2740194a3501f3ac71eedae12942c91e7b6",
        "domain": "omni",
        "question": "Tac is dressing his cat to go outside. He has four indistinguishable socks, four indistinguishable shoes, and 4 indistinguishable show-shoes. In a hurry, Tac randomly pulls pieces of clothing out of a door and tries to put them on a random one of his cat's legs; however, Tac never tries to put more than one of each type of clothing on each leg of his cat. What is the probability that, after Tac is done, the snow-shoe on each of his cat's legs is on top of the shoe, which is on top of the sock?",
        "answer": "$\\frac{1}{1296}$",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1937,
        "difficulty": 3.5
      },
      {
        "id": "0197eeb1025171b2cf1ddbcfc925972058f78911c556d32691258b1595d6efd2",
        "domain": "omni",
        "question": "Suppose $m$ and $n$ are positive integers for which the sum of the first $m$ multiples of $n$ is 120, and the sum of the first $m^{3}$ multiples of $n^{3}$ is 4032000. Determine the sum of the first $m^{2}$ multiples of $n^{2}$.",
        "answer": "20800",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2416,
        "difficulty": 5.0
      },
      {
        "id": "5446bb95c8642df2d71066c387abb620ca97fec5c6c0f7759ce03f78743b1392",
        "domain": "omni",
        "question": "An up-right path from $(a, b) \\in \\mathbb{R}^{2}$ to $(c, d) \\in \\mathbb{R}^{2}$ is a finite sequence $(x_{1}, y_{1}), \\ldots,(x_{k}, y_{k})$ of points in $\\mathbb{R}^{2}$ such that $(a, b)=(x_{1}, y_{1}),(c, d)=(x_{k}, y_{k})$, and for each $1 \\leq i<k$ we have that either $(x_{i+1}, y_{i+1})=(x_{i}+1, y_{i})$ or $(x_{i+1}, y_{i+1})=(x_{i}, y_{i}+1)$. Let $S$ be the set of all up-right paths from $(-400,-400)$ to $(400,400)$. What fraction of the paths in $S$ do not contain any point $(x, y)$ such that $|x|,|y| \\leq 10$? Express your answer as a decimal number between 0 and 1.",
        "answer": "0.2937156494680644",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 3330,
        "difficulty": 4.0
      },
      {
        "id": "76c37b92e3903cccddc90043e6d16e4a480c945887e5433cd90f7c2bb36c1881",
        "domain": "omni",
        "question": "How many pairs $(x, y)$ of non-negative integers with $0 \\leq x \\leq y$ satisfy the equation $5x^{2}-4xy+2x+y^{2}=624$?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2857,
        "difficulty": 2.5
      },
      {
        "id": "2748c337274fe54b141251b568ac75774dac714a4bd5c1091c33cebdb2acc8fc",
        "domain": "omni",
        "question": "Calculate the value of the expression $2 \\times 0 + 2 \\times 4$.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2748,
        "difficulty": 1.0
      },
      {
        "id": "b4509d0dd9bf9c37707f8e4f837a8f444efc4f369459a17e3dba0779a079bfad",
        "domain": "omni",
        "question": "Given that $a,b,c,d,e$ are real numbers such that\n$a+b+c+d+e=8$ ,\n$a^2+b^2+c^2+d^2+e^2=16$ .\nDetermine the maximum value of $e$ .",
        "answer": "\\[\n\\frac{16}{5}\n\\]",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "usamo",
        "source_index": 197,
        "difficulty": 5.0
      },
      {
        "id": "252c250e36804c460d38858dbb02864e8c47b7337c4580be60b6cc9bcf4d1f97",
        "domain": "omni",
        "question": "Seven students shared the cost of a $\\$26.00$ pizza. Each student paid either $\\$3.71$ or $\\$3.72$. How many students paid $\\$3.72$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2943,
        "difficulty": 1.0
      },
      {
        "id": "aa1cdeeb3473b142f40ad253dfe2cf94394231e75fc2eef1b13b12b71628b98f",
        "domain": "omni",
        "question": "Find the number of triples of sets $(A, B, C)$ such that: (a) $A, B, C \\subseteq\\{1,2,3, \\ldots, 8\\}$. (b) $|A \\cap B|=|B \\cap C|=|C \\cap A|=2$. (c) $|A|=|B|=|C|=4$. Here, $|S|$ denotes the number of elements in the set $S$.",
        "answer": "45360",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1073,
        "difficulty": 6.0
      },
      {
        "id": "19dc6f837ad0df47eda120796f123f99dc250604985578bd9ecc06fa5b6b8702",
        "domain": "omni",
        "question": "Begining at a vertex, an ant crawls between the vertices of a regular octahedron. After reaching a vertex, it randomly picks a neighboring vertex (sharing an edge) and walks to that vertex along the adjoining edge (with all possibilities equally likely.) What is the probability that after walking along 2006 edges, the ant returns to the vertex where it began?",
        "answer": "\\frac{2^{2005}+1}{3 \\cdot 2^{2006}}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 557,
        "difficulty": 5.25
      },
      {
        "id": "0771a15817af9310e72b2df2639915bb7c360fab6945649ed8a49e81fc2331bb",
        "domain": "omni",
        "question": "Compute the number of real solutions $(x, y, z, w)$ to the system of equations: $$\\begin{array}{rlrl} x & =z+w+z w x & z & =x+y+x y z \\\\ y & =w+x+w x y & w & =y+z+y z w \\end{array}$$",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 843,
        "difficulty": 5.0
      },
      {
        "id": "cd4ec3711e37a2268a6bdc42dcf169e0f864155d78a92d27676cc6a2300e14c9",
        "domain": "omni",
        "question": "Express, as concisely as possible, the value of the product $$\\left(0^{3}-350\\right)\\left(1^{3}-349\\right)\\left(2^{3}-348\\right)\\left(3^{3}-347\\right) \\cdots\\left(349^{3}-1\\right)\\left(350^{3}-0\\right)$$",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 3323,
        "difficulty": 4.5
      },
      {
        "id": "c9d5f67a988c1fcfb4ae9cd6993ede545303a0e305a924754c5ace521848703a",
        "domain": "omni",
        "question": "Find the number of integers $n$ such that $$ 1+\\left\\lfloor\\frac{100 n}{101}\\right\\rfloor=\\left\\lceil\\frac{99 n}{100}\\right\\rceil $$",
        "answer": "10100",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1590,
        "difficulty": 5.5
      },
      {
        "id": "bc84c8c95dd83f9e7c15106f5575620a47606a4983e0e29141dedb2916e8c376",
        "domain": "omni",
        "question": "Let $x$ and $y$ be positive real numbers and $\\theta$ an angle such that $\\theta \\neq \\frac{\\pi}{2} n$ for any integer $n$. Suppose $$\\frac{\\sin \\theta}{x}=\\frac{\\cos \\theta}{y}$$ and $$\\frac{\\cos ^{4} \\theta}{x^{4}}+\\frac{\\sin ^{4} \\theta}{y^{4}}=\\frac{97 \\sin 2 \\theta}{x^{3} y+y^{3} x}$$ Compute $\\frac{x}{y}+\\frac{y}{x}$.",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1344,
        "difficulty": 5.0
      },
      {
        "id": "f036c340966a79a3bd2ed10f386eec55a12e71adccda9bc1d68e741ceaf7fd50",
        "domain": "omni",
        "question": "Let $ABC$ be an acute triangle and let $M$ be the midpoint of $AC$. A circle $\\omega$ passing through $B$ and $M$ meets the sides $AB$ and $BC$ at points $P$ and $Q$ respectively. Let $T$ be the point such that $BPTQ$ is a parallelogram. Suppose that $T$ lies on the circumcircle of $ABC$. Determine all possible values of $\\frac{BT}{BM}$.",
        "answer": "\\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "imo_shortlist",
        "source_index": 4003,
        "difficulty": 8.0
      },
      {
        "id": "5b58b90e0ff1428842f54d2251ef5cd0b981fb56bd0f781d6e7fdad0135fdc22",
        "domain": "omni",
        "question": "Natalie has a copy of the unit interval $[0,1]$ that is colored white. She also has a black marker, and she colors the interval in the following manner: at each step, she selects a value $x \\in[0,1]$ uniformly at random, and (a) If $x \\leq \\frac{1}{2}$ she colors the interval $[x, x+\\frac{1}{2}]$ with her marker. (b) If $x>\\frac{1}{2}$ she colors the intervals $[x, 1]$ and $[0, x-\\frac{1}{2}]$ with her marker. What is the expected value of the number of steps Natalie will need to color the entire interval black?",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 944,
        "difficulty": 5.0
      },
      {
        "id": "314fdece2d9598b45bfbbcf5de0aeaf32756eea499b59c06b1c943ed2c5855ea",
        "domain": "omni",
        "question": "Bob is writing a sequence of letters of the alphabet, each of which can be either uppercase or lowercase, according to the following two rules: If he had just written an uppercase letter, he can either write the same letter in lowercase after it, or the next letter of the alphabet in uppercase. If he had just written a lowercase letter, he can either write the same letter in uppercase after it, or the preceding letter of the alphabet in lowercase. For instance, one such sequence is $a A a A B C D d c b B C$. How many sequences of 32 letters can he write that start at (lowercase) $a$ and end at (lowercase) $z$?",
        "answer": "376",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1482,
        "difficulty": 5.0
      },
      {
        "id": "ddea8db8388b97b078d1f8d0ef2605771f886e1eb06211b3bdf41559655651f8",
        "domain": "omni",
        "question": "Ewan writes out a sequence where he counts by 11s starting at 3. The resulting sequence is $3, 14, 25, 36, \\ldots$. What is a number that will appear in Ewan's sequence?",
        "answer": "113",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Sequences -> Other",
        "source": "pascal",
        "source_index": 2942,
        "difficulty": 1.5
      },
      {
        "id": "8a23967c4eae7779afdf2e72a79514ffeaed834dfd45886586f23536df69c139",
        "domain": "omni",
        "question": "Find the least possible area of a convex set in the plane that intersects both branches of the hyperbola $xy = 1$ and both branches of the hyperbola $xy = -1$. (A set $S$ in the plane is called \\emph{convex} if for any two points in $S$ the line segment connecting them is contained in $S$.)",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "putnam",
        "source_index": 3142,
        "difficulty": 7.0
      },
      {
        "id": "88a8562557419cc07764deecd9b281848987b3489c17dee6aa50410cce9839d9",
        "domain": "omni",
        "question": "An isosceles trapezoid $A B C D$ with bases $A B$ and $C D$ has $A B=13, C D=17$, and height 3. Let $E$ be the intersection of $A C$ and $B D$. Circles $\\Omega$ and $\\omega$ are circumscribed about triangles $A B E$ and $C D E$. Compute the sum of the radii of $\\Omega$ and $\\omega$.",
        "answer": "39",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 586,
        "difficulty": 5.0
      },
      {
        "id": "ae635164710a80d9a972c9fdb05f72674c9d458fe59c4c27a9c4a3001a91b223",
        "domain": "omni",
        "question": "Isabella writes the expression $\\sqrt{d}$ for each positive integer $d$ not exceeding 8 ! on the board. Seeing that these expressions might not be worth points on HMMT, Vidur simplifies each expression to the form $a \\sqrt{b}$, where $a$ and $b$ are integers such that $b$ is not divisible by the square of a prime number. (For example, $\\sqrt{20}, \\sqrt{16}$, and $\\sqrt{6}$ simplify to $2 \\sqrt{5}, 4 \\sqrt{1}$, and $1 \\sqrt{6}$, respectively.) Compute the sum of $a+b$ across all expressions that Vidur writes.",
        "answer": "534810086",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1931,
        "difficulty": 5.0
      },
      {
        "id": "8b1a6725b790986c6f387c759608deaeae8a0e7d3adb3a63a21884b478b7c2ee",
        "domain": "omni",
        "question": "Last Thursday, each of the students in M. Fermat's class brought one piece of fruit to school. Each brought an apple, a banana, or an orange. In total, $20\\%$ of the students brought an apple and $35\\%$ brought a banana. If 9 students brought oranges, how many students were in the class?",
        "answer": "20",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2749,
        "difficulty": 2.0
      },
      {
        "id": "263afbaf2effc0ae8ba4b2a6de68ddc300d8683e6162c6060dba23e22577d269",
        "domain": "omni",
        "question": "Roger the ant is traveling on a coordinate plane, starting at $(0,0)$. Every second, he moves from one lattice point to a different lattice point at distance 1, chosen with equal probability. He will continue to move until he reaches some point $P$ for which he could have reached $P$ more quickly had he taken a different route. For example, if he goes from $(0,0)$ to $(1,0)$ to $(1,1)$ to $(1,2)$ to $(0,2)$, he stops at $(0,2)$ because he could have gone from $(0,0)$ to $(0,1)$ to $(0,2)$ in only 2 seconds. The expected number of steps Roger takes before he stops can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "1103",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2060,
        "difficulty": 4.5
      },
      {
        "id": "dbc2016e367cad2c105912ce812857d2b2c6a8088f90d39c3007b28aeabd2695",
        "domain": "omni",
        "question": "Suppose $X$ is a random variable that takes on only nonnegative integer values, with $E\\left[ X \\right] = 1$, $E\\left[ X^2 \\right] = 2$, and $E \\left[ X^3 \\right] = 5$. Determine the smallest possible value of the probability of the event $X=0$.",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "putnam",
        "source_index": 3234,
        "difficulty": 7.0
      },
      {
        "id": "a0ef833ba562b722d4a7b7a812182067364b1697d59137600e46bd04829ddaae",
        "domain": "omni",
        "question": "When $5^{35}-6^{21}$ is evaluated, what is the units (ones) digit?",
        "answer": "9",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "cayley",
        "source_index": 2678,
        "difficulty": 2.5
      },
      {
        "id": "b89cb4419d03c00afc364fae1d5927053b9c74fdeb932401dc31654413526649",
        "domain": "omni",
        "question": "Find the minimum possible value of the largest of $x y, 1-x-y+x y$, and $x+y-2 x y$ if $0 \\leq x \\leq y \\leq 1$.",
        "answer": "\\frac{4}{9}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Expressions and Inequalities -> Other",
        "source": "HMMT_2",
        "source_index": 1154,
        "difficulty": 6.0
      },
      {
        "id": "a6cbb7d820793c1b6cc7347a720daea77e88505dcd91994ce42bb16b8de60d1a",
        "domain": "omni",
        "question": "During the regular season, Washington Redskins achieve a record of 10 wins and 6 losses. Compute the probability that their wins came in three streaks of consecutive wins, assuming that all possible arrangements of wins and losses are equally likely. (For example, the record LLWWWWWLWWLWWWLL contains three winning streaks, while WWWWWWWLLLLLLWWW has just two.)",
        "answer": "\\frac{315}{2002}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1483,
        "difficulty": 5.0
      },
      {
        "id": "e4782866ee0edfb843f6ae613ee1753e874ebc9384a6ae0a00979062dd62105b",
        "domain": "omni",
        "question": "Eight coins are arranged in a circle heads up. A move consists of flipping over two adjacent coins. How many different sequences of six moves leave the coins alternating heads up and tails up?",
        "answer": "7680",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1036,
        "difficulty": 5.25
      },
      {
        "id": "3dcb086ee858b5cec8ebe02e7536a06cf85ced96d32d2621befdf921942868cf",
        "domain": "omni",
        "question": "There is one odd integer \\( N \\) between 400 and 600 that is divisible by both 5 and 11. What is the sum of the digits of \\( N \\)?",
        "answer": "18",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2693,
        "difficulty": 2.5
      },
      {
        "id": "b036938dfe05c489a8ef4b93acde8ed527acebef077d84c3502bebb8a6f1dbdf",
        "domain": "omni",
        "question": "The number 3003 is the only number known to appear eight times in Pascal's triangle, at positions $\\binom{3003}{1},\\binom{3003}{3002},\\binom{a}{2},\\binom{a}{a-2},\\binom{15}{b},\\binom{15}{15-b},\\binom{14}{6},\\binom{14}{8}$. Compute $a+b(15-b)$.",
        "answer": "128",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2487,
        "difficulty": 4.0
      },
      {
        "id": "0c12b410e4334854a03e0a134a0c9a73656dd53af3d969419c1a52a81d4d2605",
        "domain": "omni",
        "question": "Circle $O$ has chord $A B$. A circle is tangent to $O$ at $T$ and tangent to $A B$ at $X$ such that $A X=2 X B$. What is \\frac{A T}{B T}?",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2074,
        "difficulty": 5.0
      },
      {
        "id": "ab97f6b23b1639012ad5d82504ed7cd4b74e0d6c6c2696ec29d76d40e0934707",
        "domain": "omni",
        "question": "Evaluate $$\\sin \\left(1998^{\\circ}+237^{\\circ}\\right) \\sin \\left(1998^{\\circ}-1653^{\\circ}\\right)$$",
        "answer": "-\\frac{1}{4}",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1431,
        "difficulty": 5.5
      },
      {
        "id": "c4363272b5e724df452f3ada4eedbab4a2cb94c96446dea2c81cb1f86c79feff",
        "domain": "omni",
        "question": "Find the minimum value of $| \\sin x + \\cos x + \\tan x + \\cot x + \\sec x + \\csc x |$ for real numbers $x$.",
        "answer": "2\\sqrt{2} - 1",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives; Mathematics -> Algebra -> Other",
        "source": "putnam",
        "source_index": 3167,
        "difficulty": 7.0
      },
      {
        "id": "cc076f98221d12038c9660acf4a24cbee1d7b1720e63ed8dbf0bcfae6b6d8a82",
        "domain": "omni",
        "question": "Let $a, b, c$ be non-negative real numbers such that $ab+bc+ca=3$. Suppose that $a^{3}b+b^{3}c+c^{3}a+2abc(a+b+c)=\\frac{9}{2}$. What is the minimum possible value of $ab^{3}+bc^{3}+ca^{3}$?",
        "answer": "18",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Abstract Algebra -> Other (Inequalities) -> Other",
        "source": "HMMT_2",
        "source_index": 480,
        "difficulty": 5.0
      },
      {
        "id": "78ad149cdfc39c259ee45158b93720630c8552e8f98985bb1b23d4c246090170",
        "domain": "omni",
        "question": "If $n$ is a positive integer such that $n^{3}+2 n^{2}+9 n+8$ is the cube of an integer, find $n$.",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1452,
        "difficulty": 4.5
      },
      {
        "id": "5e021e6b21f09ea294f22a005ced440df69de918546f6804e156503e1608c266",
        "domain": "omni",
        "question": "The polynomial $x^{3}-3 x^{2}+1$ has three real roots $r_{1}, r_{2}$, and $r_{3}$. Compute $\\sqrt[3]{3 r_{1}-2}+\\sqrt[3]{3 r_{2}-2}+\\sqrt[3]{3 r_{3}-2}$.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1925,
        "difficulty": 5.0
      },
      {
        "id": "395d8b101309454275f99453e38e1364c621a78adf6d256586138f9267a54ef2",
        "domain": "omni",
        "question": "There are two prime numbers $p$ so that $5 p$ can be expressed in the form $\\left\\lfloor\\frac{n^{2}}{5}\\right\\rfloor$ for some positive integer $n$. What is the sum of these two prime numbers?",
        "answer": "52",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1694,
        "difficulty": 4.0
      },
      {
        "id": "c471beb7d1e7de72dd909b9de5197fc40f77fe6f8904cfb3ac52b103c68dcdd9",
        "domain": "omni",
        "question": "Suppose rectangle $F O L K$ and square $L O R E$ are on the plane such that $R L=12$ and $R K=11$. Compute the product of all possible areas of triangle $R K L$.",
        "answer": "414",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2126,
        "difficulty": 4.0
      },
      {
        "id": "ce1c2aa47a429ecb6ee5abc3f3e480ee0edd7f00ee9895ccefdc34c3610d3b31",
        "domain": "omni",
        "question": "How many solid $1 \times 1 \times 1$ cubes are required to make a solid $2 \times 2 \times 2$ cube?",
        "answer": "8",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "fermat",
        "source_index": 2810,
        "difficulty": 1.5
      },
      {
        "id": "514bcba848cb5e3d816ac86eebf51f35a9ed0ac6fd59600799bb0db0e4115f41",
        "domain": "omni",
        "question": "Let $C_{k, n}$ denote the number of paths on the Cartesian plane along which you can travel from $(0,0)$ to $(k, n)$, given the following rules: 1) You can only travel directly upward or directly rightward 2) You can only change direction at lattice points 3) Each horizontal segment in the path must be at most 99 units long. Find $\\sum_{j=0}^{\\infty} C_{100 j+19,17}$",
        "answer": "100^{17}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1972,
        "difficulty": 5.5
      },
      {
        "id": "5881e5bf1296472dffebdf8bb62dd56825247dbfe0fe18a123f638de2b3af217",
        "domain": "omni",
        "question": "What is the maximum number of colours that can be used to paint an $8 \\times 8$ chessboard so that every square is painted in a single colour, and is adjacent , horizontally, vertically but not diagonally, to at least two other squares of its own colour?\n\n(A Shapovalov)",
        "answer": "16",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "ToT",
        "source_index": 3803,
        "difficulty": 5.0
      },
      {
        "id": "65c5737b5c571ba93c5cb8e917abee51712685a6137ad37b37b39a3e602c5cd5",
        "domain": "omni",
        "question": "If $u=-6$ and $x=\frac{1}{3}(3-4 u)$, what is the value of $x$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3139,
        "difficulty": 2.0
      },
      {
        "id": "8a067e004d97d69678ec2ef12316f40da8451e0c15299716226afc3d0c349499",
        "domain": "omni",
        "question": "The set of $\\{1,2,3,...,63\\}$  was divided  into three non-empty disjoint sets $A,B$. Let $a,b,c$ be the product of all numbers in each set $A,B,C$ respectively  and finally we have determined the greatest common divisor of these three products. What was the biggest result we could get?",
        "answer": "$\\boxed{2^{19} \\cdot 3^{10} \\cdot 5^{4} \\cdot 3^{3} \\cdot 11\\cdot 13\\cdot 17\\cdot 19}$",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "czech-polish-slovak matches",
        "source_index": 4077,
        "difficulty": 5.0
      },
      {
        "id": "3f91db518802664cffc8ef34e85362642ba9019a8398ccc0b9829aaa51458185",
        "domain": "omni",
        "question": "For each nonnegative integer $n$ we define $A_n = 2^{3n}+3^{6n+2}+5^{6n+2}$. Find the greatest common divisor of the numbers $A_0,A_1,\\ldots, A_{1999}$.\n\n[i]Romania[/i]",
        "answer": "7",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "junior_balkan_mo",
        "source_index": 3692,
        "difficulty": 5.0
      },
      {
        "id": "0df923003a903356dfa79a3b4aadff3fd9a3a7073fcc4ff6fbaa247249784481",
        "domain": "omni",
        "question": "If the line that passes through the points $(2,7)$ and $(a, 3a)$ has a slope of 2, what is the value of $a$?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3097,
        "difficulty": 1.5
      },
      {
        "id": "fb3a76d713599993022f6dd82c6262479f53e800cd2ee50988174a9814ee01cb",
        "domain": "omni",
        "question": "Triangle $A B C$ has $A B=4, B C=3$, and a right angle at $B$. Circles $\\omega_{1}$ and $\\omega_{2}$ of equal radii are drawn such that $\\omega_{1}$ is tangent to $A B$ and $A C, \\omega_{2}$ is tangent to $B C$ and $A C$, and $\\omega_{1}$ is tangent to $\\omega_{2}$. Find the radius of $\\omega_{1}$.",
        "answer": "\\frac{5}{7}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2403,
        "difficulty": 4.0
      },
      {
        "id": "d51d963ae755f5d44cabc47b23d7460e19adef9f1e8e53100dea47dc7aab02ea",
        "domain": "omni",
        "question": "Find out the maximum value of the numbers of edges of a solid regular octahedron that we can see from a point out of the regular octahedron.(We define we can see an edge $AB$ of the regular octahedron from point $P$ outside if and only if the intersection of non degenerate triangle $PAB$ and the solid regular octahedron is exactly edge $AB$.",
        "answer": "9",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "china_team_selection_test",
        "source_index": 46,
        "difficulty": 8.0
      },
      {
        "id": "643f4a33e2d277904c2a7fba63e277f2d01c043b3040568f0fd1f7d444668b4a",
        "domain": "omni",
        "question": "Let $A B C$ be an isosceles triangle with apex $A$. Let $I$ be the incenter. If $A I=3$ and the distance from $I$ to $B C$ is 2 , then what is the length of $B C$ ?",
        "answer": "4\\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 439,
        "difficulty": 3.5
      },
      {
        "id": "746a8391d35810c2b5934bb695cd46c73c63a25c2fbaa8f182af9bfae38d356b",
        "domain": "omni",
        "question": "Dewa writes down a list of four integers. He calculates the average of each group of three of the four integers. These averages are $32,39,40,44$. What is the largest of the four integers?",
        "answer": "59",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3009,
        "difficulty": 1.5
      },
      {
        "id": "849a4eb500493e77d63a46a65a0b9c1f4f82a6045e5fef7d9570ce89ac0f68d4",
        "domain": "omni",
        "question": "A function, $f$, has $f(2)=5$ and $f(3)=7$. In addition, $f$ has the property that $f(m)+f(n)=f(mn)$ for all positive integers $m$ and $n$. What is the value of $f(12)$?",
        "answer": "17",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "fermat",
        "source_index": 3051,
        "difficulty": 2.5
      },
      {
        "id": "e42c046cea9b6dc76cb982e317de526aaca43d344225ce12b4a23d2658469ca7",
        "domain": "omni",
        "question": "The lines $y=x, y=2 x$, and $y=3 x$ are the three medians of a triangle with perimeter 1. Find the length of the longest side of the triangle.",
        "answer": "\\sqrt{\\frac{\\sqrt{58}}{2+\\sqrt{34}+\\sqrt{58}}}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 630,
        "difficulty": 5.25
      },
      {
        "id": "51f5a8f1ca0ed10621765bb958963788a7c9597bbe5f19dfede2e25c999d1a40",
        "domain": "omni",
        "question": "Find the value of $$\\sum_{a=1}^{\\infty} \\sum_{b=1}^{\\infty} \\sum_{c=1}^{\\infty} \\frac{a b(3 a+c)}{4^{a+b+c}(a+b)(b+c)(c+a)}$$",
        "answer": "\\frac{1}{54}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "HMMT_2",
        "source_index": 1568,
        "difficulty": 4.5
      },
      {
        "id": "97c57b55bfa98db292d8b6dd2f96cc3357611343ec1918ebf198184ba2e3ebf5",
        "domain": "omni",
        "question": "Let $S$ be a set, $|S|=35$. A set $F$ of mappings from $S$ to itself is called to be satisfying property $P(k)$, if for any $x,y\\in S$, there exist $f_1, \\cdots, f_k \\in F$ (not necessarily different), such that $f_k(f_{k-1}(\\cdots (f_1(x))))=f_k(f_{k-1}(\\cdots (f_1(y))))$.\nFind the least positive integer $m$, such that if $F$ satisfies property $P(2019)$, then it also satisfies property $P(m)$.",
        "answer": "595",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Logic",
        "source": "china_national_olympiad",
        "source_index": 25,
        "difficulty": 8.0
      },
      {
        "id": "c6ef14acf7a7354bce12139ffe490986f94f5f0eb7a3b1ae9b3669ebc9ce2f3f",
        "domain": "omni",
        "question": "You are given a $10 \\times 2$ grid of unit squares. Two different squares are adjacent if they share a side. How many ways can one mark exactly nine of the squares so that no two marked squares are adjacent?",
        "answer": "36",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1401,
        "difficulty": 4.0
      },
      {
        "id": "5ddd956d76e2e341744b7059dc889b44dba9262873613a42bf5f6e9266d219f6",
        "domain": "omni",
        "question": "If the perimeter of a square is 28, what is the side length of the square?",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "pascal",
        "source_index": 2915,
        "difficulty": 1.0
      },
      {
        "id": "339b7b2894e6224c47b0d4e3b943735be0db3d45519fcfecbc074d99537c6b78",
        "domain": "omni",
        "question": "Consider sequences \\(a\\) of the form \\(a=\\left(a_{1}, a_{2}, \\ldots, a_{20}\\right)\\) such that each term \\(a_{i}\\) is either 0 or 1. For each such sequence \\(a\\), we can produce a sequence \\(b=\\left(b_{1}, b_{2}, \\ldots, b_{20}\\right)\\), where \\(b_{i}= \\begin{cases}a_{i}+a_{i+1} & i=1 \\\\ a_{i-1}+a_{i}+a_{i+1} & 1<i<20 \\\\ a_{i-1}+a_{i} & i=20\\end{cases}\\). How many sequences \\(b\\) are there that can be produced by more than one distinct sequence \\(a\\)?",
        "answer": "64",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2249,
        "difficulty": 5.0
      },
      {
        "id": "bbaacb81eacc3b3db3f1588dc66ac284b44e0eedc90bc3fecd7f16b38c4557e8",
        "domain": "omni",
        "question": "Let $a, b, c, d, e$ be nonnegative integers such that $625 a+250 b+100 c+40 d+16 e=15^{3}$. What is the maximum possible value of $a+b+c+d+e$ ?",
        "answer": "153",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1222,
        "difficulty": 5.0
      },
      {
        "id": "b978cf2d3076e66e7291878340903cedff3144be37f48e7416dcc9dd8a19fb37",
        "domain": "omni",
        "question": "In the game of Galactic Dominion, players compete to amass cards, each of which is worth a certain number of points. Say you are playing a version of this game with only two kinds of cards, planet cards and hegemon cards. Each planet card is worth 2010 points, and each hegemon card is worth four points per planet card held. You start with no planet cards and no hegemon cards, and, on each turn, starting at turn one, you take either a planet card or a hegemon card, whichever is worth more points given the hand you currently hold. Define a sequence $\\left\\{a_{n}\\right\\}$ for all positive integers $n$ by setting $a_{n}$ to be 0 if on turn $n$ you take a planet card and 1 if you take a hegemon card. What is the smallest value of $N$ such that the sequence $a_{N}, a_{N+1}, \\ldots$ is necessarily periodic (meaning that there is a positive integer $k$ such that $a_{n+k}=a_{n}$ for all $\\left.n \\geq N\\right)$ ?",
        "answer": "503",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2521,
        "difficulty": 5.0
      },
      {
        "id": "87fb7b2ed30cabfef2e1ea423dcd4533646bacee8803fa542cc54fbd1c2f24bc",
        "domain": "omni",
        "question": "Sindy writes down the positive integers less than 200 in increasing order, but skips the multiples of 10. She then alternately places + and - signs before each of the integers, yielding an expression $+1-2+3-4+5-6+7-8+9-11+12-\\cdots-199$. What is the value of the resulting expression?",
        "answer": "-100",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 1759,
        "difficulty": 3.5
      },
      {
        "id": "99bf16db6cb4b0bc61d158d5ccf0093a322591cd55a3c496f7dd508b81ab0559",
        "domain": "omni",
        "question": "Let $d_n$ be the determinant of the $n \\times n$ matrix whose entries, from\nleft to right and then from top to bottom, are $\\cos 1, \\cos 2, \\dots, \\cos\nn^2$. Evaluate\n$\\lim_{n\\to\\infty} d_n$.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Matrices",
        "source": "putnam",
        "source_index": 3148,
        "difficulty": 7.0
      },
      {
        "id": "f1d65d5c63138ba37ad0942d82896fd36a1fff9e20a523dea6331b753159c4e4",
        "domain": "omni",
        "question": "A point in three-space has distances $2,6,7,8,9$ from five of the vertices of a regular octahedron. What is its distance from the sixth vertex?",
        "answer": "\\sqrt{21}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 504,
        "difficulty": 4.5
      },
      {
        "id": "ea52f2ec0c8135996304a63c26ff24e1f7340101b557eb9ff2627ac28f0f54d3",
        "domain": "omni",
        "question": "For any positive integer $n$, let $\\tau(n)$ denote the number of positive divisors of $n$. If $n$ is a positive integer such that $\\frac{\\tau\\left(n^{2}\\right)}{\\tau(n)}=3$, compute $\\frac{\\tau\\left(n^{7}\\right)}{\\tau(n)}$.",
        "answer": "29",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1480,
        "difficulty": 5.0
      },
      {
        "id": "1e717852574a149e87f3620b97213631d36f938ccc2ca2ce151aad032830ba95",
        "domain": "omni",
        "question": "For how many ordered triples $(a, b, c)$ of positive integers are the equations $abc+9=ab+bc+ca$ and $a+b+c=10$ satisfied?",
        "answer": "21",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 483,
        "difficulty": 4.0
      },
      {
        "id": "2c63974d56fbba18e63d363359a0cff26f8a95ad6b2026540d86e85b791fc50f",
        "domain": "omni",
        "question": "A [i]permutation[/i] of the set of positive integers $[n] = \\{1, 2, . . . , n\\}$ is a sequence $(a_1 , a_2 , \\ldots, a_n ) $ such that each element of $[n]$ appears precisely one time as a term of the sequence. For example, $(3, 5, 1, 2, 4)$ is a permutation of $[5]$. Let $P (n)$ be the number of permutations of $[n]$ for which $ka_k$ is a perfect square for all $1 \\leq k \\leq n$. Find with proof the smallest $n$ such that $P (n)$ is a multiple of $2010$.",
        "answer": "4489",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usajmo",
        "source_index": 3709,
        "difficulty": 5.5
      },
      {
        "id": "2e3c673ec8056b4b557a111d1ce3e9fbbb074009bc87b64c2b4f32ec5f0c633d",
        "domain": "omni",
        "question": "A set of 6 distinct lattice points is chosen uniformly at random from the set $\\{1,2,3,4,5,6\\}^{2}$. Let $A$ be the expected area of the convex hull of these 6 points. Estimate $N=\\left\\lfloor 10^{4} A\\right\\rfloor$. An estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{4}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
        "answer": "104552",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 916,
        "difficulty": 5.0
      },
      {
        "id": "8db15258764ff178e379f29c1817b463d53a107a83dd43a9bc99edfc1152df25",
        "domain": "omni",
        "question": "Let $a=\\sqrt{17}$ and $b=i \\sqrt{19}$, where $i=\\sqrt{-1}$. Find the maximum possible value of the ratio $|a-z| /|b-z|$ over all complex numbers $z$ of magnitude 1 (i.e. over the unit circle $|z|=1$ ).",
        "answer": "\\frac{4}{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 639,
        "difficulty": 5.25
      },
      {
        "id": "54e7555815c7e1009a2bda55aeb5139143157eca914ac859b61943025ef76225",
        "domain": "omni",
        "question": "Hagrid has 100 animals. Among these animals, each is either striped or spotted but not both, each has either wings or horns but not both, there are 28 striped animals with wings, there are 62 spotted animals, and there are 36 animals with horns. How many of Hagrid's spotted animals have horns?",
        "answer": "26",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2955,
        "difficulty": 1.5
      },
      {
        "id": "5fe05ba2a5b9978d67475fbb7c7aae75b69315bb939dcf6ce02ec5888b2a57bf",
        "domain": "omni",
        "question": "Let $\\mathcal{P}$ be a parabola, and let $V_{1}$ and $F_{1}$ be its vertex and focus, respectively. Let $A$ and $B$ be points on $\\mathcal{P}$ so that $\\angle AV_{1}B=90^{\\circ}$. Let $\\mathcal{Q}$ be the locus of the midpoint of $AB$. It turns out that $\\mathcal{Q}$ is also a parabola, and let $V_{2}$ and $F_{2}$ denote its vertex and focus, respectively. Determine the ratio $F_{1}F_{2}/V_{1}V_{2}$.",
        "answer": "\\frac{7}{8}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Curves -> Other",
        "source": "HMMT_2",
        "source_index": 627,
        "difficulty": 5.25
      },
      {
        "id": "ed159d4dfb9d41a24eaec9bd285ad9dc82ab95fd6fbf16b7bf5a9ef299898e3a",
        "domain": "omni",
        "question": "Patrick and Anderson are having a snowball fight. Patrick throws a snowball at Anderson which is shaped like a sphere with a radius of 10 centimeters. Anderson catches the snowball and uses the snow from the snowball to construct snowballs with radii of 4 centimeters. Given that the total volume of the snowballs that Anderson constructs cannot exceed the volume of the snowball that Patrick threw, how many snowballs can Anderson construct?",
        "answer": "15",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1013,
        "difficulty": 5.0
      },
      {
        "id": "77a3e3c173bfc9648caf5838e90353b99f05dcf2426ff529ebe7814705982323",
        "domain": "omni",
        "question": "If $a, b$, and $c$ are random real numbers from 0 to 1, independently and uniformly chosen, what is the average (expected) value of the smallest of $a, b$, and $c$?",
        "answer": "1/4",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 965,
        "difficulty": 4.0
      },
      {
        "id": "20fa13a9680e03ee5f5fe3c2a534f530efe9486c3461ddd7be669f5d2b2dd4f8",
        "domain": "omni",
        "question": "Suppose that $k>0$ and that the line with equation $y=3kx+4k^{2}$ intersects the parabola with equation $y=x^{2}$ at points $P$ and $Q$. If $O$ is the origin and the area of $\triangle OPQ$ is 80, then what is the slope of the line?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "fermat",
        "source_index": 2758,
        "difficulty": 2.5
      },
      {
        "id": "c6452a10016b73065a29d5598ef6a265edd08d436b0d6413225d7d6280b54309",
        "domain": "omni",
        "question": "In the Year 0 of Cambridge there is one squirrel and one rabbit. Both animals multiply in numbers quickly. In particular, if there are $m$ squirrels and $n$ rabbits in Year $k$, then there will be $2 m+2019$ squirrels and $4 n-2$ rabbits in Year $k+1$. What is the first year in which there will be strictly more rabbits than squirrels?",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 832,
        "difficulty": 4.0
      },
      {
        "id": "4b4add1f86d375d75d49de5e5dc4c55f18dbb1818b884b084d0a68f966b5c1d4",
        "domain": "omni",
        "question": "Let $A B C D$ be a convex quadrilateral whose diagonals $A C$ and $B D$ meet at $P$. Let the area of triangle $A P B$ be 24 and let the area of triangle $C P D$ be 25 . What is the minimum possible area of quadrilateral $A B C D ?$",
        "answer": "49+20 \\sqrt{6}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2563,
        "difficulty": 4.5
      },
      {
        "id": "2e9cc967460290271f737eca6307344c01adb6f24aa47baed35d2796cf51af4f",
        "domain": "omni",
        "question": "Two players play alternately on a $ 5 \\times 5$ board. The first player always enters a $ 1$ into an empty square and the second player always enters a $ 0$ into an empty square. When the board is full, the sum of the numbers in each of the nine $ 3 \\times 3$ squares is calculated and the first player's score is the largest such sum. What is the largest score the first player can make, regardless of the responses of the second player?",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 3994,
        "difficulty": 7.0
      },
      {
        "id": "af4f0ff09d6bc802eca2b664a383ef0af500a603177643c54c375ba8c99de9f8",
        "domain": "omni",
        "question": "Mayar and Rosie are 90 metres apart. Starting at the same time, they run towards each other. Mayar runs twice as fast as Rosie. How far has Mayar run when they meet?",
        "answer": "60",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3479,
        "difficulty": 1.5
      },
      {
        "id": "22acc0dab6818376163781757c0028a0c7a879109ff70c0d8b2a7f522ddf67b2",
        "domain": "omni",
        "question": "For any positive integer $n$, let $f(n)$ denote the number of 1's in the base-2 representation of $n$. For how many values of $n$ with $1 \\leq n \\leq 2002$ do we have $f(n)=f(n+1)$?",
        "answer": "501",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1208,
        "difficulty": 5.0
      },
      {
        "id": "ca0ce8d11a196bb7cfa103850eda68acd4cc094396ff3bf6e1981c838787fd7d",
        "domain": "omni",
        "question": "Find the number of ways to choose two nonempty subsets $X$ and $Y$ of $\\{1,2, \\ldots, 2001\\}$, such that $|Y|=1001$ and the smallest element of $Y$ is equal to the largest element of $X$.",
        "answer": "2^{2000}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1843,
        "difficulty": 5.25
      },
      {
        "id": "58a50bfe6a14d56fe3e7aa26c39fb10d1c0a2361a651bc359847df4e4761af45",
        "domain": "omni",
        "question": "If $10^n = 1000^{20}$, what is the value of $n$?",
        "answer": "60",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "fermat",
        "source_index": 2795,
        "difficulty": 2.0
      },
      {
        "id": "c9bfd954c079de503f86b58a7bbc6cb9796465d476bad2b56e939a0630c87af0",
        "domain": "omni",
        "question": "Suppose there are 100 cookies arranged in a circle, and 53 of them are chocolate chip, with the remainder being oatmeal. Pearl wants to choose a contiguous subsegment of exactly 67 cookies and wants this subsegment to have exactly \\(k\\) chocolate chip cookies. Find the sum of the \\(k\\) for which Pearl is guaranteed to succeed regardless of how the cookies are arranged.",
        "answer": "71",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1562,
        "difficulty": 5.0
      },
      {
        "id": "b7733aa7d9f014bb2c710cb9547be2fb94977bdacc57e9dd24176b47ea180591",
        "domain": "omni",
        "question": "Let $P(x)$ be the monic polynomial with rational coefficients of minimal degree such that $\\frac{1}{\\sqrt{2}}$, $\\frac{1}{\\sqrt{3}}, \\frac{1}{\\sqrt{4}}, \\ldots, \\frac{1}{\\sqrt{1000}}$ are roots of $P$. What is the sum of the coefficients of $P$?",
        "answer": "\\frac{1}{16000}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 772,
        "difficulty": 4.0
      },
      {
        "id": "83eef7f357d961e12a8f0fc7e1db3f517d075f999b6f6fe2b8b55607ed6d3c44",
        "domain": "omni",
        "question": "Calculate the value of the expression $\\frac{1+(3 \\times 5)}{2}$.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2930,
        "difficulty": 1.0
      },
      {
        "id": "2833650846ec3fa305896fd2a2ce9fa1d0c2d2d1304b966b1cdf7637954e5812",
        "domain": "omni",
        "question": "Determine which of the following numbers is smallest in value: $54 \\sqrt{3}, 144,108 \\sqrt{6}-108 \\sqrt{2}$.",
        "answer": "$54 \\sqrt{3}$",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "HMMT_11",
        "source_index": 1932,
        "difficulty": 5.0
      },
      {
        "id": "793ff429d8af362ab0118b168e0848a61993842e9e5b94d109becea0bd7ee68b",
        "domain": "omni",
        "question": "David and Evan each repeatedly flip a fair coin. David will stop when he flips a tail, and Evan will stop once he flips 2 consecutive tails. Find the probability that David flips more total heads than Evan.",
        "answer": "\\frac{1}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2240,
        "difficulty": 5.25
      },
      {
        "id": "cf27129a2382a0767553882c64abc510a71e3bf7b5165ddf9bf0aa22687df07f",
        "domain": "omni",
        "question": "A rectangle is divided into four smaller rectangles, labelled W, X, Y, and Z. The perimeters of rectangles W, X, and Y are 2, 3, and 5, respectively. What is the perimeter of rectangle Z?",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "fermat",
        "source_index": 3169,
        "difficulty": 2.5
      },
      {
        "id": "706f0baae8a640a3d77e7d6af1af483c54de1733a744f64188f1c5dd84c3ea27",
        "domain": "omni",
        "question": "Marisa has a collection of $2^{8}-1=255$ distinct nonempty subsets of $\\{1,2,3,4,5,6,7,8\\}$. For each step she takes two subsets chosen uniformly at random from the collection, and replaces them with either their union or their intersection, chosen randomly with equal probability. (The collection is allowed to contain repeated sets.) She repeats this process $2^{8}-2=254$ times until there is only one set left in the collection. What is the expected size of this set?",
        "answer": "\\frac{1024}{255}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2346,
        "difficulty": 4.0
      },
      {
        "id": "bff864066334cb52eac6b4e844fa0d8f8c0de4fb7b25a33758e3540cb3007f1d",
        "domain": "omni",
        "question": "Let $t_{n}$ equal the integer closest to $\\sqrt{n}$. What is the sum $\\frac{1}{t_{1}}+\\frac{1}{t_{2}}+\\frac{1}{t_{3}}+\\frac{1}{t_{4}}+\\cdots+\\frac{1}{t_{2008}}+\\frac{1}{t_{2009}}+\\frac{1}{t_{2010}}$?",
        "answer": "88 \\frac{2}{3}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "fermat",
        "source_index": 3223,
        "difficulty": 2.5
      },
      {
        "id": "27397d390a0f364e4fbdc853a1d7e7b69a9ec49844f078e7a18c51cc52eef955",
        "domain": "omni",
        "question": "In a square of side length 4 , a point on the interior of the square is randomly chosen and a circle of radius 1 is drawn centered at the point. What is the probability that the circle intersects the square exactly twice?",
        "answer": "\\frac{\\pi+8}{16}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1901,
        "difficulty": 4.0
      },
      {
        "id": "b9a1344421120fbecd7190b8d683f7ed8ad72f84fcc4be6511e44b2ef15ec055",
        "domain": "omni",
        "question": "Let $m, n > 2$ be integers. One of the angles of a regular $n$-gon is dissected into $m$ angles of equal size by $(m-1)$ rays. If each of these rays intersects the polygon again at one of its vertices, we say $n$ is $m$-cut. Compute the smallest positive integer $n$ that is both 3-cut and 4-cut.",
        "answer": "14",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1035,
        "difficulty": 4.0
      },
      {
        "id": "b98b26d28fe08e3668e0f9694bc2b027a9d943a6fd956d51cb2584cf5d778305",
        "domain": "omni",
        "question": "Let $\\mathcal{H}$ be a regular hexagon with side length one. Peter picks a point $P$ uniformly and at random within $\\mathcal{H}$, then draws the largest circle with center $P$ that is contained in $\\mathcal{H}$. What is this probability that the radius of this circle is less than $\\frac{1}{2}$?",
        "answer": "\\frac{2 \\sqrt{3}-1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2070,
        "difficulty": 5.0
      },
      {
        "id": "2cb09856754ffa83d4dc16a40b5f0bbf4960abb766c7b0bb1226ff810ef1448e",
        "domain": "omni",
        "question": "In a gumball machine containing 13 red, 5 blue, 1 white, and 9 green gumballs, what is the least number of gumballs that must be bought to guarantee receiving 3 gumballs of the same color?",
        "answer": "8",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "fermat",
        "source_index": 2774,
        "difficulty": 2.5
      },
      {
        "id": "d11caff5cc2b2726d00946519484157628d5954111e1ef091193efe5b636c69e",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. Let $I_{A}, I_{B}, I_{C}$ be the $A, B, C$ excenters of this triangle, and let $O$ be the circumcenter of the triangle. Let $\\gamma_{A}, \\gamma_{B}, \\gamma_{C}$ be the corresponding excircles and $\\omega$ be the circumcircle. $X$ is one of the intersections between $\\gamma_{A}$ and $\\omega$. Likewise, $Y$ is an intersection of $\\gamma_{B}$ and $\\omega$, and $Z$ is an intersection of $\\gamma_{C}$ and $\\omega$. Compute $$\\cos \\angle O X I_{A}+\\cos \\angle O Y I_{B}+\\cos \\angle O Z I_{C}$$",
        "answer": "-\\frac{49}{65}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1072,
        "difficulty": 5.0
      },
      {
        "id": "943aa843b43a208fa30a2cd3892e2950342e9ceb1bcb0da1f682d248019b7d8d",
        "domain": "omni",
        "question": "An empty $2020 \\times 2020 \\times 2020$ cube is given, and a $2020 \\times 2020$ grid of square unit cells is drawn on each of its six faces. A beam is a $1 \\times 1 \\times 2020$ rectangular prism. Several beams are placed inside the cube subject to the following conditions:\nThe two faces of each beam coincide with unit cells lying on opposite faces of the cube. (Hence, there are possible positions for a beam.)  No two beams have intersecting interiors.  The interiors of each of the four faces of each beam touch either a face of the cube or the interior of the face of another beam.\nWhat is the smallest positive number of beams that can be placed to satisfy these conditions?",
        "answer": "\\[\n3030\n\\]",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "usamo",
        "source_index": 391,
        "difficulty": 8.0
      },
      {
        "id": "ec4bd8e44be8b7a15b5966820e50e5cf67ed473bf8e3ff78e82f6c8085077b76",
        "domain": "omni",
        "question": "Lily has a $300 \\times 300$ grid of squares. She now removes $100 \\times 100$ squares from each of the four corners and colors each of the remaining 50000 squares black and white. Given that no $2 \\times 2$ square is colored in a checkerboard pattern, find the maximum possible number of (unordered) pairs of squares such that one is black, one is white and the squares share an edge.",
        "answer": "49998",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 296,
        "difficulty": 5.0
      },
      {
        "id": "ea9483ab98fa1d330bb4e2a7be8fa1372e184570cd6b643851a164a26271043c",
        "domain": "omni",
        "question": "What is the value of \\((-1)^{3}+(-1)^{2}+(-1)\\)?",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2665,
        "difficulty": 1.5
      },
      {
        "id": "a947c6b046444fd8dc2620b73bf2d53548831b2d5d70474d63da4559a9031ea9",
        "domain": "omni",
        "question": "Compute $$\\lim _{A \\rightarrow+\\infty} \\frac{1}{A} \\int_{1}^{A} A^{\\frac{1}{x}} \\mathrm{~d} x$$",
        "answer": "1",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals",
        "source": "imc",
        "source_index": 2579,
        "difficulty": 8.0
      },
      {
        "id": "873ac223b0aebd57d737feb591ce2ef69f0b4568209064c70cf4d7b3fc217274",
        "domain": "omni",
        "question": "A palindrome is a positive integer that reads the same backwards as forwards, such as 82328. What is the smallest 5 -digit palindrome that is a multiple of 99 ?",
        "answer": "54945",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 534,
        "difficulty": 3.5
      },
      {
        "id": "e86ba2f6a4b5c599118ce43fa5d6fb42859032f6556f63095d9b0ac91b3bbe63",
        "domain": "omni",
        "question": "If $x=2018$, what is the value of the expression $x^{2}+2x-x(x+1)$?",
        "answer": "2018",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2669,
        "difficulty": 1.5
      },
      {
        "id": "d50564777dc140696f0a574b73e1ef094a41ce2c99c14e55eb57fdbcb9e03981",
        "domain": "omni",
        "question": "An ant starts at one vertex of a tetrahedron. Each minute it walks along a random edge to an adjacent vertex. What is the probability that after one hour the ant winds up at the same vertex it started at?",
        "answer": "\\frac{3^{59}+1}{4 \\cdot 3^{59}}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1064,
        "difficulty": 5.25
      },
      {
        "id": "3247bdcfd162500c6cb827f3dd4613e873e0fd3706bbbb00973d55017cddecc4",
        "domain": "omni",
        "question": "Let $p_{i}$ be the $i$th prime. Let $$f(x)=\\sum_{i=1}^{50} p_{i} x^{i-1}=2+3x+\\cdots+229x^{49}$$ If $a$ is the unique positive real number with $f(a)=100$, estimate $A=\\lfloor 100000a\\rfloor$. An estimate of $E$ will earn $\\max (0,\\lfloor 20-|A-E| / 250\\rfloor)$ points.",
        "answer": "83601",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2622,
        "difficulty": 3.5
      },
      {
        "id": "3e4b40cd62f928b7c6673e2e2cea8ba9fe986a3bb11164744043c5d348b38819",
        "domain": "omni",
        "question": "The value of $\\sqrt{3^{3}+3^{3}+3^{3}}$ is what?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3456,
        "difficulty": 1.0
      },
      {
        "id": "3be336845c4e3eba95a93522ce6f57f5a3740ee482f7b22cc1304e5912f8b4ec",
        "domain": "omni",
        "question": "Snacks are purchased for 17 soccer players. Juice boxes come in packs of 3 and cost $2.00 per pack. Apples come in bags of 5 and cost $4.00 per bag. What is the minimum amount of money that Danny spends to ensure every player gets a juice box and an apple?",
        "answer": "\\$28.00",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 3173,
        "difficulty": 2.5
      },
      {
        "id": "29090d3fd6c948e9a2bc8a83b094bb160249c0acbfb7b705a2ebb8cdaeaa2631",
        "domain": "omni",
        "question": "In Mrs. Warner's class, there are 30 students. Strangely, 15 of the students have a height of 1.60 m and 15 of the students have a height of 1.22 m. Mrs. Warner lines up \\(n\\) students so that the average height of any four consecutive students is greater than 1.50 m and the average height of any seven consecutive students is less than 1.50 m. What is the largest possible value of \\(n\\)?",
        "answer": "9",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 3062,
        "difficulty": 2.5
      },
      {
        "id": "ae4c594c9100788275355e0cfdcda2d8c21dc02c110ec7db7185c49dcdd73a7f",
        "domain": "omni",
        "question": "Compute the number of complex numbers $z$ with $|z|=1$ that satisfy $$1+z^{5}+z^{10}+z^{15}+z^{18}+z^{21}+z^{24}+z^{27}=0$$",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 859,
        "difficulty": 5.25
      },
      {
        "id": "d58f4b7b1d101a7358d04b6c253896639db9386c8531e62ddfe8edd2e112dbec",
        "domain": "omni",
        "question": "A 50-card deck consists of 4 cards labeled \" $i$ \" for $i=1,2, \\ldots, 12$ and 2 cards labeled \" 13 \". If Bob randomly chooses 2 cards from the deck without replacement, what is the probability that his 2 cards have the same label?",
        "answer": "\\frac{73}{1225}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2273,
        "difficulty": 4.0
      },
      {
        "id": "4492c324d4e442226628b8d0f5f026cbf266d2cd1d4a5fbd9be55e1d934787fc",
        "domain": "omni",
        "question": "Find the sum $\\sum_{d=1}^{2012}\\left\\lfloor\\frac{2012}{d}\\right\\rfloor$.",
        "answer": "15612",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2392,
        "difficulty": 4.5
      },
      {
        "id": "e77e8ac11de8df6b322387f0241c26af0eac435823aa368cb4404e44a3a2fa4d",
        "domain": "omni",
        "question": "A hockey team has 6 more red helmets than blue helmets. The ratio of red helmets to blue helmets is $5:3$. What is the total number of red helmets and blue helmets?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2950,
        "difficulty": 1.5
      },
      {
        "id": "fd27b0b56139e7bc53f9805fc333a99905119d9cb455fe3fef920b936d7b5272",
        "domain": "omni",
        "question": "Define the sequence $\\{x_{i}\\}_{i \\geq 0}$ by $x_{0}=2009$ and $x_{n}=-\\frac{2009}{n} \\sum_{k=0}^{n-1} x_{k}$ for all $n \\geq 1$. Compute the value of $\\sum_{n=0}^{2009} 2^{n} x_{n}$",
        "answer": "2009",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1447,
        "difficulty": 6.0
      },
      {
        "id": "3bad97abc8ef11a3f00a337c5abee66a9ba8c0ee1fddea5113f4c0d508e66833",
        "domain": "omni",
        "question": "Compute the number of ways to color the vertices of a regular heptagon red, green, or blue (with rotations and reflections distinct) such that no isosceles triangle whose vertices are vertices of the heptagon has all three vertices the same color.",
        "answer": "294",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1801,
        "difficulty": 5.25
      },
      {
        "id": "a770211d2a8af75c10f722cd8b377abd838852340f7576a2f157eb343a41f60e",
        "domain": "omni",
        "question": "For a positive integer $n$, denote by $\\tau(n)$ the number of positive integer divisors of $n$, and denote by $\\phi(n)$ the number of positive integers that are less than or equal to $n$ and relatively prime to $n$. Call a positive integer $n$ good if $\\varphi(n)+4 \\tau(n)=n$. For example, the number 44 is good because $\\varphi(44)+4 \\tau(44)=44$. Find the sum of all good positive integers $n$.",
        "answer": "172",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1024,
        "difficulty": 4.5
      },
      {
        "id": "0cd280e355e10b6ec46e73bd454bbad1f02c3c726a51dad35a506f5e5b9487d3",
        "domain": "omni",
        "question": "Consider a permutation $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ of $\\{1,2,3,4,5\\}$. We say the tuple $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ is flawless if for all $1 \\leq i<j<k \\leq 5$, the sequence $\\left(a_{i}, a_{j}, a_{k}\\right)$ is not an arithmetic progression (in that order). Find the number of flawless 5-tuples.",
        "answer": "20",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1777,
        "difficulty": 4.5
      },
      {
        "id": "bd2abf3a3e6cd63bd0d631dcad684c9321544e7751d11beff30383566a524b15",
        "domain": "omni",
        "question": "A sequence is defined by $a_{0}=1$ and $a_{n}=2^{a_{n-1}}$ for $n \\geq 1$. What is the last digit (in base 10) of $a_{15}$?",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 1125,
        "difficulty": 5.0
      },
      {
        "id": "c6f6230827fa219f9de46d95a715c629c53c0be645468880ac3c96d56d993228",
        "domain": "omni",
        "question": "Find the sum of squares of all distinct complex numbers $x$ satisfying the equation $0=4 x^{10}-7 x^{9}+5 x^{8}-8 x^{7}+12 x^{6}-12 x^{5}+12 x^{4}-8 x^{3}+5 x^{2}-7 x+4$",
        "answer": "-\\frac{7}{16}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 664,
        "difficulty": 5.0
      },
      {
        "id": "ac159572eba04470ae3fd51a190bfbfd2d0bec423db86a870270e348ebc63c61",
        "domain": "omni",
        "question": "The function $f(x)$ is of the form $a x^{2}+b x+c$ for some integers $a, b$, and $c$. Given that $\\{f(177883), f(348710), f(796921), f(858522)\\} = \\{1324754875645,1782225466694,1984194627862,4388794883485\\}$ compute $a$.",
        "answer": "23",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1414,
        "difficulty": 5.0
      },
      {
        "id": "e15192fd326ebdad1162de711deff5edb0821fe1bf2a8b5493f9bc9c6f2c74be",
        "domain": "omni",
        "question": "What is the 7th oblong number?",
        "answer": "56",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2726,
        "difficulty": 2.0
      },
      {
        "id": "da4df0fe51f9b152899db1c06f332b12e8bfdffbd5a31245baa2178599f8d153",
        "domain": "omni",
        "question": "Suppose $x$ is a real number such that $\\sin \\left(1+\\cos ^{2} x+\\sin ^{4} x\\right)=\\frac{13}{14}$. Compute $\\cos \\left(1+\\sin ^{2} x+\\cos ^{4} x\\right)$.",
        "answer": "-\\frac{3 \\sqrt{3}}{14}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1391,
        "difficulty": 6.0
      },
      {
        "id": "0e70a0a907e0e07a8d8b6902a62acd14b6a23a323f978afdff0d70d796fb5c3e",
        "domain": "omni",
        "question": "What is 30% of 200?",
        "answer": "60",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals",
        "source": "pascal",
        "source_index": 2881,
        "difficulty": 1.0
      },
      {
        "id": "3f24f32cc8a59395df3ceadfb2a223720098bb8a13f79b2eff256bde39f58d16",
        "domain": "omni",
        "question": "What is \\( 110\\% \\) of 500?",
        "answer": "550",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3025,
        "difficulty": 1.5
      },
      {
        "id": "6a340a3c21c6336a53ee6cab951dba46b95523f9add4e5cbf188159ffc985819",
        "domain": "omni",
        "question": "On the blackboard, Amy writes 2017 in base-$a$ to get $133201_{a}$. Betsy notices she can erase a digit from Amy's number and change the base to base-$b$ such that the value of the number remains the same. Catherine then notices she can erase a digit from Betsy's number and change the base to base-$c$ such that the value still remains the same. Compute, in decimal, $a+b+c$.",
        "answer": "22",
        "topic": "Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1958,
        "difficulty": 4.0
      },
      {
        "id": "f11d8a3d527714577b8e7c62b64d6b5a9fcc63b905fcef448785848d51563845",
        "domain": "omni",
        "question": "Let $w, x, y$, and $z$ be positive real numbers such that $0 \\neq \\cos w \\cos x \\cos y \\cos z$, $2 \\pi =w+x+y+z$, $3 \\tan w =k(1+\\sec w)$, $4 \\tan x =k(1+\\sec x)$, $5 \\tan y =k(1+\\sec y)$, $6 \\tan z =k(1+\\sec z)$. Find $k$.",
        "answer": "\\sqrt{19}",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 698,
        "difficulty": 5.0
      },
      {
        "id": "4edf5e3f799a9f88d97961585dcd5302dc647036886b58309572bfca826ab42a",
        "domain": "omni",
        "question": "Find the number of subsets $S$ of $\\{1,2, \\ldots, 48\\}$ satisfying both of the following properties: - For each integer $1 \\leq k \\leq 24$, exactly one of $2 k-1$ and $2 k$ is in $S$. - There are exactly nine integers $1 \\leq m \\leq 47$ so that both $m$ and $m+1$ are in $S$.",
        "answer": "177100",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2524,
        "difficulty": 5.0
      },
      {
        "id": "e5b88e689e9f658788770f073a47418b7f8c56290d5fc6d566931bd18f4d7895",
        "domain": "omni",
        "question": "If Alex does not sing on Saturday, then she has a $70 \\%$ chance of singing on Sunday; however, to rest her voice, she never sings on both days. If Alex has a $50 \\%$ chance of singing on Sunday, find the probability that she sings on Saturday.",
        "answer": "\\frac{2}{7}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1102,
        "difficulty": 4.0
      },
      {
        "id": "fd4021fb2980b203a630e8991f55c40e669989e88f33de7767e7fea7896917d4",
        "domain": "omni",
        "question": "Carl and Bob can demolish a building in 6 days, Anne and Bob can do it in 3, Anne and Carl in 5. How many days does it take all of them working together if Carl gets injured at the end of the first day and can't come back? Express your answer as a fraction in lowest terms.",
        "answer": "\\frac{59}{20}",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1430,
        "difficulty": 5.5
      },
      {
        "id": "9956d0ba0ea4433859720a3d1e0eab290bda4a16b3bf54166600bb60b8cb7cde",
        "domain": "omni",
        "question": "Compute the number of nonempty subsets $S \\subseteq\\{-10,-9,-8, \\ldots, 8,9,10\\}$ that satisfy $|S|+\\min (S)$. $\\max (S)=0$.",
        "answer": "335",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 1713,
        "difficulty": 5.0
      },
      {
        "id": "7262608d9a531dacce31912196b8301c54905584e5f01c037287358315c3f0a8",
        "domain": "omni",
        "question": "What is the largest possible radius of a circle contained in a 4-dimensional hypercube of side length 1?",
        "answer": "\\frac{\\sqrt{2}}{2}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Other",
        "source": "putnam",
        "source_index": 3232,
        "difficulty": 8.0
      },
      {
        "id": "5ef9678dcee143fcfa8f53849d3943ad89f219d28500a88eeb166282c9bbc9f6",
        "domain": "omni",
        "question": "Suppose that $R, S$ and $T$ are digits and that $N$ is the four-digit positive integer $8 R S T$. That is, $N$ has thousands digit 8, hundreds digit $R$, tens digits $S$, and ones (units) digit $T$, which means that $N=8000+100 R+10 S+T$. Suppose that the following conditions are all true: - The two-digit integer $8 R$ is divisible by 3. - The three-digit integer $8 R S$ is divisible by 4. - The four-digit integer $8 R S T$ is divisible by 5. - The digits of $N$ are not necessarily all different. What is the number of possible values for the integer $N$?",
        "answer": "14",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3066,
        "difficulty": 2.0
      },
      {
        "id": "c1dd29fceec107d15b0d94daf3b60a1ecbfa198de6a812fecc6771e0450bba12",
        "domain": "omni",
        "question": "Find the volume of the set of points $(x, y, z)$ satisfying $$\\begin{array}{r} x, y, z \\geq 0 \\\\ x+y \\leq 1 \\\\ y+z \\leq 1 \\\\ z+x \\leq 1 \\end{array}$$",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume",
        "source": "HMMT_2",
        "source_index": 2344,
        "difficulty": 4.0
      },
      {
        "id": "5d4df269281910b86d9f44e736b50a54e22b68a34e986c0032e1882013d5c256",
        "domain": "omni",
        "question": "Which of the following is equal to $110 \\%$ of 500?",
        "answer": "550",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3473,
        "difficulty": 1.0
      },
      {
        "id": "ea69a210303d0a59878ac4449403407e1dc5c9cdfda1b55059e21d7002d210de",
        "domain": "omni",
        "question": "On the number line, points $M$ and $N$ divide $L P$ into three equal parts. What is the value at $M$?",
        "answer": "\\frac{1}{9}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3049,
        "difficulty": 1.0
      },
      {
        "id": "385b3a6101901e4b5d383bc65b58438a977d95c4f4d11062ed413a8700493dd9",
        "domain": "omni",
        "question": "Let $a, b, c, d$ be real numbers such that $a^{2}+b^{2}+c^{2}+d^{2}=1$. Determine the minimum value of $(a-b)(b-c)(c-d)(d-a)$ and determine all values of $(a, b, c, d)$ such that the minimum value is achieved.",
        "answer": "-\\frac{1}{8}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Calculus -> Applications of Derivatives (for optimization purposes) -> Other",
        "source": "apmoapmo_sol",
        "source_index": 1476,
        "difficulty": 7.0
      },
      {
        "id": "4f550d221f6f32b02702cdc34a4fb5a60764046d1e141594424364c442ecbd0b",
        "domain": "omni",
        "question": "Compute the positive real number $x$ satisfying $x^{\\left(2 x^{6}\\right)}=3$",
        "answer": "\\sqrt[6]{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1620,
        "difficulty": 5.5
      },
      {
        "id": "8962e3008b1ea2a0da1d43c5ed91cfe61a73c8cdeeed65e4a36404f0cb074d98",
        "domain": "omni",
        "question": "The quadrilateral $ABCD$ has the following equality $\\angle ABC=\\angle BCD=150^{\\circ}$. Moreover, $AB=18$ and $BC=24$, the equilateral triangles $\\triangle APB,\\triangle BQC,\\triangle CRD$ are drawn outside the quadrilateral. If $P(X)$ is the perimeter of the polygon $X$, then the following equality is true $P(APQRD)=P(ABCD)+32$. Determine the length of the side $CD$.",
        "answer": "10",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "all_levels",
        "source_index": 3747,
        "difficulty": 8.0
      },
      {
        "id": "3fcb3c87beda488f0ff24877d124bdf1e92ac94a2319cc6b1716704d06a20fb1",
        "domain": "omni",
        "question": "In triangle $BCD$, $\\angle CBD=\\angle CDB$ because $BC=CD$. If $\\angle BCD=80+50+30=160$, find $\\angle CBD=\\angle CDB$.",
        "answer": "10",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2359,
        "difficulty": 1.5
      },
      {
        "id": "4eb221a3ec90c1590ab1a9ca98226711688dd17ea386abbf419414eb3b5b8249",
        "domain": "omni",
        "question": "Kelvin and 15 other frogs are in a meeting, for a total of 16 frogs. During the meeting, each pair of distinct frogs becomes friends with probability $\\frac{1}{2}$. Kelvin thinks the situation after the meeting is cool if for each of the 16 frogs, the number of friends they made during the meeting is a multiple of 4. Say that the probability of the situation being cool can be expressed in the form $\\frac{a}{b}$, where $a$ and $b$ are relatively prime. Find $a$.",
        "answer": "1167",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other; Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1526,
        "difficulty": 7.0
      },
      {
        "id": "eda3dec880f14cf50fa70eadfacb380c25b081325abf7472defa368023271528",
        "domain": "omni",
        "question": "How many of the integers from 1 to 100, inclusive, have at least one digit equal to 6?",
        "answer": "19",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2973,
        "difficulty": 2.0
      },
      {
        "id": "b0a87eccb6854cd3b81cc6f4df9005b5fce8c6a64e499dabecfe3a3ed86ad837",
        "domain": "omni",
        "question": "Several positive integers are given, not necessarily all different. Their sum is 2003. Suppose that $n_{1}$ of the given numbers are equal to $1, n_{2}$ of them are equal to $2, \\ldots, n_{2003}$ of them are equal to 2003. Find the largest possible value of $$n_{2}+2 n_{3}+3 n_{4}+\\cdots+2002 n_{2003}$$",
        "answer": "2002",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1296,
        "difficulty": 4.0
      },
      {
        "id": "f7a6d9b10fff3da028c3af8be91b29e5cad761502363cef8ad080f38d92fa6a8",
        "domain": "omni",
        "question": "Determine all positive integers relatively prime to all the terms of the infinite sequence \\[ a_n=2^n+3^n+6^n -1,\\ n\\geq 1. \\]",
        "answer": "1",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Congruences",
        "source": "imo",
        "source_index": 3829,
        "difficulty": 7.5
      },
      {
        "id": "1e2214856cacd0dc0448850916146f202429553ca23365d6249c3e6a633fea97",
        "domain": "omni",
        "question": "Find the minimum possible value of $\\sqrt{58-42 x}+\\sqrt{149-140 \\sqrt{1-x^{2}}}$ where $-1 \\leq x \\leq 1$",
        "answer": "\\sqrt{109}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_11",
        "source_index": 2116,
        "difficulty": 6.0
      },
      {
        "id": "76ca86db7782da8facbe5ff8ca839a70737aaa61591096d88df4a1e8a1b88777",
        "domain": "omni",
        "question": "Which number from the set $\\{1,2,3,4,5,6,7,8,9,10,11\\}$ must be removed so that the mean (average) of the numbers remaining in the set is 6.1?",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3450,
        "difficulty": 1.5
      },
      {
        "id": "789ec2e7f53f15c2a4ddad6afb5cc834cb7ab040480123de47bbbf27be0d4931",
        "domain": "omni",
        "question": "At Wednesday's basketball game, the Cayley Comets scored 90 points. At Friday's game, they scored $80\\%$ as many points as they scored on Wednesday. How many points did they score on Friday?",
        "answer": "72",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2689,
        "difficulty": 1.0
      },
      {
        "id": "c68aa9d77830eddd39d4d893fd33368b6a683070035630f4d0d809e8eec11925",
        "domain": "omni",
        "question": "Find the number of ways in which the nine numbers $$1,12,123,1234, \\ldots, 123456789$$ can be arranged in a row so that adjacent numbers are relatively prime.",
        "answer": "0",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2536,
        "difficulty": 5.0
      },
      {
        "id": "96ee0db283aa1cefb58abd79b40a447f35c183c2632ad3daa5a004e0186ec393",
        "domain": "omni",
        "question": "Pentagon $S P E A K$ is inscribed in triangle $N O W$ such that $S$ and $P$ lie on segment $N O, K$ and $A$ lie on segment $N W$, and $E$ lies on segment $O W$. Suppose that $N S=S P=P O$ and $N K=K A=A W$. Given that $E P=E K=5$ and $E A=E S=6$, compute $O W$.",
        "answer": "\\frac{3 \\sqrt{610}}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2277,
        "difficulty": 4.5
      },
      {
        "id": "4aae0013af8de1c4e8f8f2a7c2bb5f825ded217ce909b548a4067d574736595f",
        "domain": "omni",
        "question": "Consider pairs $(f,g)$ of functions from the set of nonnegative integers to itself such that \n[list]\n[*]$f(0) \\geq f(1) \\geq f(2) \\geq \\dots \\geq f(300) \\geq 0$\n[*]$f(0)+f(1)+f(2)+\\dots+f(300) \\leq 300$\n[*]for any 20 nonnegative integers $n_1, n_2, \\dots, n_{20}$, not necessarily distinct, we have $$g(n_1+n_2+\\dots+n_{20}) \\leq f(n_1)+f(n_2)+\\dots+f(n_{20}).$$\n[/list]\nDetermine the maximum possible value of $g(0)+g(1)+\\dots+g(6000)$ over all such pairs of functions.\n\n[i]Sean Li[/i]",
        "answer": "115440",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "usa_team_selection_test_for_imo",
        "source_index": 20,
        "difficulty": 8.0
      },
      {
        "id": "7adad32caaf9aad775eedb9a740518f8dcd5817aafdfa01e00df0a61411bd14b",
        "domain": "omni",
        "question": "Let $N$ be the number of functions $f$ from $\\{1,2, \\ldots, 101\\} \\rightarrow\\{1,2, \\ldots, 101\\}$ such that $f^{101}(1)=2$. Find the remainder when $N$ is divided by 103.",
        "answer": "43",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2439,
        "difficulty": 4.0
      },
      {
        "id": "b2be23ae7b3387450af29d050cae4ad524d9ed2ca8878a54cb0f1e3b5cc815a2",
        "domain": "omni",
        "question": "Find the sum of all positive integers $n$ such that there exists an integer $b$ with $|b| \\neq 4$ such that the base -4 representation of $n$ is the same as the base $b$ representation of $n$.",
        "answer": "1026",
        "topic": "Mathematics -> Number Theory -> Base Representations -> Other",
        "source": "HMMT_11",
        "source_index": 2069,
        "difficulty": 4.5
      },
      {
        "id": "0f16c94182bccbf5110e27f294ae59538b9f41aadd72b8564ca984e5d417732b",
        "domain": "omni",
        "question": "Find the sum of the absolute values of the roots of $x^{4}-4 x^{3}-4 x^{2}+16 x-8=0$.",
        "answer": "2+2\\sqrt{2}+2\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1351,
        "difficulty": 5.5
      },
      {
        "id": "1ab8c3fe45a3ca0e72df051f71c26d932beb4c721b98c23d33f022948093f2bd",
        "domain": "omni",
        "question": "Suppose $(a_{1}, a_{2}, a_{3}, a_{4})$ is a 4-term sequence of real numbers satisfying the following two conditions: - $a_{3}=a_{2}+a_{1}$ and $a_{4}=a_{3}+a_{2}$ - there exist real numbers $a, b, c$ such that $a n^{2}+b n+c=\\cos \\left(a_{n}\\right)$ for all $n \\in\\{1,2,3,4\\}$. Compute the maximum possible value of $\\cos \\left(a_{1}\\right)-\\cos \\left(a_{4}\\right)$ over all such sequences $(a_{1}, a_{2}, a_{3}, a_{4})$.",
        "answer": "-9+3\\sqrt{13}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives (finite differences and interpolation) -> Other; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1387,
        "difficulty": 6.0
      },
      {
        "id": "85a3c692d6bd7e97bee79639ccc50866cff215dbb7a2cc255bd3eced08bf9b5e",
        "domain": "omni",
        "question": "Frank and Joe are playing ping pong. For each game, there is a $30 \\%$ chance that Frank wins and a $70 \\%$ chance Joe wins. During a match, they play games until someone wins a total of 21 games. What is the expected value of number of games played per match?",
        "answer": "30",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 362,
        "difficulty": 5.0
      },
      {
        "id": "e909a518994b0e36e210f26467315fa50a5d5088b405a504224bc9152d3f6154",
        "domain": "omni",
        "question": "After viewing the John Harvard statue, a group of tourists decides to estimate the distances of nearby locations on a map by drawing a circle, centered at the statue, of radius $\\sqrt{n}$ inches for each integer $2020 \\leq n \\leq 10000$, so that they draw 7981 circles altogether. Given that, on the map, the Johnston Gate is 10 -inch line segment which is entirely contained between the smallest and the largest circles, what is the minimum number of points on this line segment which lie on one of the drawn circles? (The endpoint of a segment is considered to be on the segment.)",
        "answer": "49",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2313,
        "difficulty": 4.0
      },
      {
        "id": "eef53745020bdf10e2956410c359a76d7b42c9707777600dadfdd9999de7d6a5",
        "domain": "omni",
        "question": "Manya has a stack of $85=1+4+16+64$ blocks comprised of 4 layers (the $k$ th layer from the top has $4^{k-1}$ blocks). Each block rests on 4 smaller blocks, each with dimensions half those of the larger block. Laura removes blocks one at a time from this stack, removing only blocks that currently have no blocks on top of them. Find the number of ways Laura can remove precisely 5 blocks from Manya's stack (the order in which they are removed matters).",
        "answer": "3384",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1192,
        "difficulty": 4.5
      },
      {
        "id": "0e37d840ce7035c1221e606736819f3e701da069ef380960b19b9279fdc66bb1",
        "domain": "omni",
        "question": "Alex and Bob have 30 matches. Alex picks up somewhere between one and six matches (inclusive), then Bob picks up somewhere between one and six matches, and so on. The player who picks up the last match wins. How many matches should Alex pick up at the beginning to guarantee that he will be able to win?",
        "answer": "2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 415,
        "difficulty": 3.5
      },
      {
        "id": "00ca1330a40be55c9368317ddeaadb15bcc3064fc0fc5ee64397cc763c41d525",
        "domain": "omni",
        "question": "Sally the snail sits on the $3 \\times 24$ lattice of points $(i, j)$ for all $1 \\leq i \\leq 3$ and $1 \\leq j \\leq 24$. She wants to visit every point in the lattice exactly once. In a move, Sally can move to a point in the lattice exactly one unit away. Given that Sally starts at $(2,1)$, compute the number of possible paths Sally can take.",
        "answer": "4096",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 1139,
        "difficulty": 4.0
      },
      {
        "id": "efd30b924e473486c3c27b2e1f745c51a5e44e3ba61a3b8e849eed0c06c17afe",
        "domain": "omni",
        "question": "3. Consider all 100-digit positive integers such that each decimal digit of these equals $2,3,4,5,6$, or 7 . How many of these integers are divisible by $2^{100}$ ?\n\t\n\tPavel Kozhevnikov",
        "answer": "3^{100}",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "ToT",
        "source_index": 3943,
        "difficulty": 6.5
      },
      {
        "id": "e3c18cd49eb8d99e13a2babc247c0facbe27bee814a668f972e39d3bc5fd740d",
        "domain": "omni",
        "question": "The Athenas are playing a 44 game season. They have 20 wins and 15 losses so far. What is the smallest number of their remaining games that they must win to make the playoffs, given they must win at least 60% of all of their games?",
        "answer": "7",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3103,
        "difficulty": 1.5
      },
      {
        "id": "55821e7de28ab5f4e3c0e9bae90358d3d5867178b84e20a9feb485fec821c511",
        "domain": "omni",
        "question": "Determine the greatest real number $ C $, such that for every positive integer $ n\\ge 2 $, there exists $ x_1, x_2,..., x_n  \\in [-1,1]$, so that\n$$\\prod_{1\\le i<j\\le n}(x_i-x_j) \\ge C^{\\frac{n(n-1)}{2}}$$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "china_team_selection_test",
        "source_index": 76,
        "difficulty": 6.0
      },
      {
        "id": "4bd4baffddb2937a4eef8803f31a786cf04e939a31bd300c253febb7178a9a5d",
        "domain": "omni",
        "question": "Alice is thinking of a positive real number $x$, and Bob is thinking of a positive real number $y$. Given that $x^{\\sqrt{y}}=27$ and $(\\sqrt{x})^{y}=9$, compute $x y$.",
        "answer": "16 \\sqrt[4]{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 2225,
        "difficulty": 4.5
      },
      {
        "id": "67d43bd5253fb19a49d854989a9b8c6cee99f83b1e4106567d004423a4f33c9f",
        "domain": "omni",
        "question": "Our next object up for bid is an arithmetic progression of primes. For example, the primes 3,5, and 7 form an arithmetic progression of length 3. What is the largest possible length of an arithmetic progression formed of positive primes less than 1,000,000? Be prepared to justify your answer.",
        "answer": "12",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1149,
        "difficulty": 5.0
      },
      {
        "id": "0ef446319ca240894fc30aba2c9c7ad9d3ddba357a4df4dc0dd6cb99220946e5",
        "domain": "omni",
        "question": "Farmer James has some strange animals. His hens have 2 heads and 8 legs, his peacocks have 3 heads and 9 legs, and his zombie hens have 6 heads and 12 legs. Farmer James counts 800 heads and 2018 legs on his farm. What is the number of animals that Farmer James has on his farm?",
        "answer": "203",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1887,
        "difficulty": 3.5
      },
      {
        "id": "49ec2b6d9bb46ecae7c48b1fdb97b3c7888f3b2bdb4da0cca202fdeb84f64957",
        "domain": "omni",
        "question": "Let $P$ be a polynomial such that $P(x)=P(0)+P(1) x+P(2) x^{2}$ and $P(-1)=1$. Compute $P(3)$.",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 726,
        "difficulty": 5.0
      },
      {
        "id": "fa29a4f8c138defafc83b0c14cffba5c27eaca2ec72c7d87f65ff3d7e54e1bff",
        "domain": "omni",
        "question": "Find the smallest integer $n \\geq 5$ for which there exists a set of $n$ distinct pairs $\\left(x_{1}, y_{1}\\right), \\ldots,\\left(x_{n}, y_{n}\\right)$ of positive integers with $1 \\leq x_{i}, y_{i} \\leq 4$ for $i=1,2, \\ldots, n$, such that for any indices $r, s \\in\\{1,2, \\ldots, n\\}$ (not necessarily distinct), there exists an index $t \\in\\{1,2, \\ldots, n\\}$ such that 4 divides $x_{r}+x_{s}-x_{t}$ and $y_{r}+y_{s}-y_{t}$.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_2",
        "source_index": 729,
        "difficulty": 5.25
      },
      {
        "id": "96190ca5984f1a301377a716fe988362ab89e75a262458b88e3da51abeec3a74",
        "domain": "omni",
        "question": "Let acute triangle $ABC$ have circumcenter $O$, and let $M$ be the midpoint of $BC$. Let $P$ be the unique point such that $\\angle BAP=\\angle CAM, \\angle CAP=\\angle BAM$, and $\\angle APO=90^{\\circ}$. If $AO=53, OM=28$, and $AM=75$, compute the perimeter of $\\triangle BPC$.",
        "answer": "192",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 875,
        "difficulty": 5.25
      },
      {
        "id": "c30226be662ed705d110c3b4e906361b87c21a3b661bd68c2e32f54de0c3134c",
        "domain": "omni",
        "question": "Let $P$ be the set of points $$\\{(x, y) \\mid 0 \\leq x, y \\leq 25, x, y \\in \\mathbb{Z}\\}$$ and let $T$ be the set of triangles formed by picking three distinct points in $P$ (rotations, reflections, and translations count as distinct triangles). Compute the number of triangles in $T$ that have area larger than 300.",
        "answer": "436",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 961,
        "difficulty": 5.0
      },
      {
        "id": "4c8a8ea41191150fc71c50359cfeb9d0b57058e114de4eeb729aaa4efc116a05",
        "domain": "omni",
        "question": "An equilateral triangle lies in the Cartesian plane such that the $x$-coordinates of its vertices are pairwise distinct and all satisfy the equation $x^{3}-9 x^{2}+10 x+5=0$. Compute the side length of the triangle.",
        "answer": "2 \\sqrt{17}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1658,
        "difficulty": 3.5
      },
      {
        "id": "8c46b6568ed3dc49896c096636c0fc66d4d0444810f479ea3be1e9e283758b5c",
        "domain": "omni",
        "question": "There are $2022$ users on a social network called Mathbook, and some of them are Mathbook-friends. (On Mathbook, friendship is always mutual and permanent.) \n\nStarting now, Mathbook will only allow a new friendship to be formed between two users if they have [i]at least two[/i] friends in common. What is the minimum number of friendships that must already exist so that every user could eventually become friends with every other user?",
        "answer": "3031",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "usamo",
        "source_index": 3573,
        "difficulty": 8.0
      },
      {
        "id": "29bd969871a6548002bc3006c411912289ceaf8c74ac44da1356d7f3a9bb42c9",
        "domain": "omni",
        "question": "Find the smallest $n$ such that $n!$ ends with 10 zeroes.",
        "answer": "45",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 2365,
        "difficulty": 3.5
      },
      {
        "id": "2e28794b3dd6d3bf410a8c2a5d74725859929ee09eb29df3f4c5351c3a887f28",
        "domain": "omni",
        "question": "There are 10 people who want to choose a committee of 5 people among them. They do this by first electing a set of $1,2,3$, or 4 committee leaders, who then choose among the remaining people to complete the 5-person committee. In how many ways can the committee be formed, assuming that people are distinguishable? (Two committees that have the same members but different sets of leaders are considered to be distinct.)",
        "answer": "7560",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1218,
        "difficulty": 4.0
      },
      {
        "id": "61186ec9919973db016d68fad7f5d7adfe90dccac0cb32a3f55db0a56393ea11",
        "domain": "omni",
        "question": "There is a unique quadruple of positive integers $(a, b, c, k)$ such that $c$ is not a perfect square and $a+\\sqrt{b+\\sqrt{c}}$ is a root of the polynomial $x^{4}-20 x^{3}+108 x^{2}-k x+9$. Compute $c$.",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1832,
        "difficulty": 4.5
      },
      {
        "id": "152f6173af503d68f20e8b503f1ecb5c397a3305fd79c2e789d6e5e83fb81ca3",
        "domain": "omni",
        "question": "Compute the number of quadruples $(a, b, c, d)$ of positive integers satisfying $12a+21b+28c+84d=2024$.",
        "answer": "2024",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 689,
        "difficulty": 5.0
      },
      {
        "id": "749273e08a04a224a76d7cd470c8e9f4f474943aa0168bcb9a0aeefefedc92c9",
        "domain": "omni",
        "question": "In how many ways can the set of ordered pairs of integers be colored red and blue such that for all $a$ and $b$, the points $(a, b),(-1-b, a+1)$, and $(1-b, a-1)$ are all the same color?",
        "answer": "16",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1021,
        "difficulty": 5.0
      },
      {
        "id": "0e7102dadedf748be6481e0fdbf3ba2aafa19ea33092b8aa338cf367800e3e73",
        "domain": "omni",
        "question": "Jerry has ten distinguishable coins, each of which currently has heads facing up. He chooses one coin and flips it over, so it now has tails facing up. Then he picks another coin (possibly the same one as before) and flips it over. How many configurations of heads and tails are possible after these two flips?",
        "answer": "46",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1995,
        "difficulty": 3.5
      },
      {
        "id": "d40aabc754745f2ffb13004211745fb7f4426eb78f66022cd115b2764e16461b",
        "domain": "omni",
        "question": "Determine the greatest positive integer $k$ that satisfies the following property: The set of positive integers can be partitioned into $k$ subsets $A_1, A_2, \\ldots, A_k$ such that for all integers $n \\geq 15$ and all $i \\in \\{1, 2, \\ldots, k\\}$ there exist two distinct elements of $A_i$ whose sum is $n.$\n\n[i]",
        "answer": "3",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Other",
        "source": "imo_shortlist",
        "source_index": 3993,
        "difficulty": 9.0
      },
      {
        "id": "4aa9feb583be727a8393dae62355a46c4e3c6b0dcfe8c7c0f144baede23cd60d",
        "domain": "omni",
        "question": "Let $f(n)$ be the number of times you have to hit the $\\sqrt{ }$ key on a calculator to get a number less than 2 starting from $n$. For instance, $f(2)=1, f(5)=2$. For how many $1<m<2008$ is $f(m)$ odd?",
        "answer": "242",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1220,
        "difficulty": 4.0
      },
      {
        "id": "0d2667239b56fe3be3ce9a679cbd15550bc6c536f2e980e4e5bcb644bf7a65a7",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=4, B C=8$, and $C A=5$. Let $M$ be the midpoint of $B C$, and let $D$ be the point on the circumcircle of $A B C$ so that segment $A D$ intersects the interior of $A B C$, and $\\angle B A D=\\angle C A M$. Let $A D$ intersect side $B C$ at $X$. Compute the ratio $A X / A D$.",
        "answer": "$\\frac{9}{41}$",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1988,
        "difficulty": 5.25
      },
      {
        "id": "256c818824634576556e19bc61a35fae2b0ed468df9976c1d0c38860bf284fcb",
        "domain": "omni",
        "question": "If $\\left(a+\\frac{1}{a}\\right)^{2}=3$, find $\\left(a+\\frac{1}{a}\\right)^{3}$ in terms of $a$.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 379,
        "difficulty": 3.5
      },
      {
        "id": "4a54545fc2ae0f2a22845303e2a8f082e8a2905bca88638c407a3cdbb1ac9675",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $\\angle B=90^{\\circ}$. Given that there exists a point $D$ on $A C$ such that $A D=D C$ and $B D=B C$, compute the value of the ratio $\\frac{A B}{B C}$.",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other",
        "source": "HMMT_11",
        "source_index": 2196,
        "difficulty": 4.0
      },
      {
        "id": "14455da7ae43ece35a3166f83f594d09ce04ea54a5e3a36a0ded39c78525ef23",
        "domain": "omni",
        "question": "A cube has an edge length of 30. A rectangular solid has edge lengths 20, 30, and $L$. If the cube and the rectangular solid have equal surface areas, what is the value of $L$?",
        "answer": "42",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Surface Area",
        "source": "pascal",
        "source_index": 2933,
        "difficulty": 2.0
      },
      {
        "id": "2d1939445e07736f2e7a7e68d492c0a3e9c062d54658aa5d44bb9af54e7812b1",
        "domain": "omni",
        "question": "The point $P$ is inside of an equilateral triangle with side length $10$ so that the distance from $P$ to two of the sides are $1$ and $3$. Find the distance from $P$ to the third side.",
        "answer": "5\\sqrt{3} - 4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "jbmo_shortlists",
        "source_index": 3745,
        "difficulty": 4.5
      },
      {
        "id": "8cb7db22fc72da267b3c2bea7d3f96c4cd503749c26df6ed300dfe76954627b9",
        "domain": "omni",
        "question": "Evaluate the expression $2x^{2}+3x^{2}$ when $x=2$.",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "fermat",
        "source_index": 2826,
        "difficulty": 1.0
      },
      {
        "id": "9c38014e9c830b665e6e11d49011c7575c4b524744102eaf36f47e9114a99298",
        "domain": "omni",
        "question": "The average of a set of distinct primes is 27. What is the largest prime that can be in this set?",
        "answer": "139",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1490,
        "difficulty": 5.0
      },
      {
        "id": "b126b7cef2fe1b317d84ad80e29c4b9d5758f3ff4dc21d382c9d025aefbe5a93",
        "domain": "omni",
        "question": "If the number of zeros in the integer equal to $(10^{100}) \times (100^{10})$ is sought, what is this number?",
        "answer": "120",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3104,
        "difficulty": 1.5
      },
      {
        "id": "ce772ad93988588148677cb9ce58ef7fed49542c820121d9496a6de750e823d5",
        "domain": "omni",
        "question": "Let $x=2001^{1002}-2001^{-1002}$ and $y=2001^{1002}+2001^{-1002}$. Find $x^{2}-y^{2}$.",
        "answer": "-4",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 405,
        "difficulty": 4.0
      },
      {
        "id": "2fbc8534bfeabd3fc8e6a05dfa47dff9e0658e44690bea99dfff550d2b3dda2c",
        "domain": "omni",
        "question": "If $\frac{x-y}{z-y}=-10$, what is the value of $\frac{x-z}{y-z}$?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3274,
        "difficulty": 2.5
      },
      {
        "id": "353c11ebe03575f36e0c2adc24de7c977a9019da1f645b4258a5b4d21b34e7a0",
        "domain": "omni",
        "question": "Alice draws three cards from a standard 52-card deck with replacement. Ace through 10 are worth 1 to 10 points respectively, and the face cards King, Queen, and Jack are each worth 10 points. The probability that the sum of the point values of the cards drawn is a multiple of 10 can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "26597",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2512,
        "difficulty": 5.25
      },
      {
        "id": "41e844d171f90d1fa25832e080796e9932d2601dd116ac419c039a4db613b1df",
        "domain": "omni",
        "question": "Let $A B C D$ be a rectangle, and let $E$ and $F$ be points on segment $A B$ such that $A E=E F=F B$. If $C E$ intersects the line $A D$ at $P$, and $P F$ intersects $B C$ at $Q$, determine the ratio of $B Q$ to $C Q$.",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1830,
        "difficulty": 4.5
      },
      {
        "id": "ed7a22c73975d93fb76dfe93cb7aac2125326f1e7107affd64d1273d2ef3e618",
        "domain": "omni",
        "question": "Let $a_1,a_2,a_3,\\cdots$ be a non-decreasing sequence of positive integers. For $m\\ge1$ , define $b_m=\\min\\{n: a_n \\ge m\\}$ , that is, $b_m$ is the minimum value of $n$ such that $a_n\\ge m$ . If $a_{19}=85$ , determine the maximum value of $a_1+a_2+\\cdots+a_{19}+b_1+b_2+\\cdots+b_{85}$ .",
        "answer": "\\boxed{1700}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "usamo",
        "source_index": 179,
        "difficulty": 7.0
      },
      {
        "id": "72d955cd62f03f3d01e362038718a9ce4ea2697d4389e55c2f833a90e0d2716c",
        "domain": "omni",
        "question": "Contessa is taking a random lattice walk in the plane, starting at $(1,1)$. (In a random lattice walk, one moves up, down, left, or right 1 unit with equal probability at each step.) If she lands on a point of the form $(6 m, 6 n)$ for $m, n \\in \\mathbb{Z}$, she ascends to heaven, but if she lands on a point of the form $(6 m+3,6 n+3)$ for $m, n \\in \\mathbb{Z}$, she descends to hell. What is the probability that she ascends to heaven?",
        "answer": "\\frac{13}{22}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1094,
        "difficulty": 4.5
      },
      {
        "id": "34849a4cd2b1b0c3a4c8be17b7537e0e433b673240ff9b623f31e4606038fec5",
        "domain": "omni",
        "question": "Let $b(n)$ be the number of digits in the base -4 representation of $n$. Evaluate $\\sum_{i=1}^{2013} b(i)$.",
        "answer": "12345",
        "topic": "Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2137,
        "difficulty": 4.0
      },
      {
        "id": "ee37b7e1ea0a4042ad63948d4326fe412fac487cc39563ce603ff77517ff23e8",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, \\ldots$ be a sequence defined by $a_{1}=a_{2}=1$ and $a_{n+2}=a_{n+1}+a_{n}$ for $n \\geq 1$. Find $$\\sum_{n=1}^{\\infty} \\frac{a_{n}}{4^{n+1}}$$",
        "answer": "\\frac{1}{11}",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series",
        "source": "HMMT_2",
        "source_index": 1508,
        "difficulty": 5.0
      },
      {
        "id": "f30adda718e28c91504555f6b44bbc424603b947f6242afa73a42ef46fca6bf7",
        "domain": "omni",
        "question": "Calculate the sum of the coefficients of $P(x)$ if $\\left(20 x^{27}+2 x^{2}+1\\right) P(x)=2001 x^{2001}$.",
        "answer": "87",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 932,
        "difficulty": 5.0
      },
      {
        "id": "14fef4e84b84fb22e652db9605ce99d8442db78f86bb0a5e5c11058f508902a7",
        "domain": "omni",
        "question": "The smallest of nine consecutive integers is 2012. These nine integers are placed in the circles to the right. The sum of the three integers along each of the four lines is the same. If this sum is as small as possible, what is the value of $u$?",
        "answer": "2015",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "pascal",
        "source_index": 2909,
        "difficulty": 1.5
      },
      {
        "id": "37557f17b8e3246fdadec7d9d15f7556fd91e207f88c3a8b2cf8c9eabbfe9801",
        "domain": "omni",
        "question": "The numbers $1,2, \\ldots, 20$ are put into a hat. Claire draws two numbers from the hat uniformly at random, $a<b$, and then puts them back into the hat. Then, William draws two numbers from the hat uniformly at random, $c<d$. Let $N$ denote the number of integers $n$ that satisfy exactly one of $a \\leq n \\leq b$ and $c \\leq n \\leq d$. Compute the probability $N$ is even.",
        "answer": "\\frac{181}{361}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 696,
        "difficulty": 5.0
      },
      {
        "id": "4cf39a9ca810490c7e82963dfdf47325c857a910ca26cbbb71645359157633f0",
        "domain": "omni",
        "question": "Determine the maximum value of $m^2+n^2$, where $m$ and $n$ are integers in the range $1,2,\\ldots,1981$ satisfying $(n^2-mn-m^2)^2=1$.",
        "answer": "\\boxed{987^2+1597^2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "imo",
        "source_index": 4159,
        "difficulty": 5.5
      },
      {
        "id": "f4bf046923876431a0ce993445e65f3ed43d2c67cdc26f857262b1d0ca8033a6",
        "domain": "omni",
        "question": "You are trapped in ancient Japan, and a giant enemy crab is approaching! You must defeat it by cutting off its two claws and six legs and attacking its weak point for massive damage. You cannot cut off any of its claws until you cut off at least three of its legs, and you cannot attack its weak point until you have cut off all of its claws and legs. In how many ways can you defeat the giant enemy crab?",
        "answer": "14400",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1681,
        "difficulty": 5.25
      },
      {
        "id": "eba366b236de045a347f0d3a21a7986a7f58a60b3662ffdb3e195a886d82b772",
        "domain": "omni",
        "question": "Find $\\sum_{k=0}^{\\infty}\\left\\lfloor\\frac{1+\\sqrt{\\frac{2000000}{4^{k}}}}{2}\\right\\rfloor$ where $\\lfloor x\\rfloor$ denotes the largest integer less than or equal to $x$.",
        "answer": "1414",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 626,
        "difficulty": 4.0
      },
      {
        "id": "d44d8c82c8093d4453de8dc0cb25e247a0f48af4591fd834de67716def949001",
        "domain": "omni",
        "question": "If $n$ is a positive integer, the notation $n$! (read \" $n$ factorial\") is used to represent the product of the integers from 1 to $n$. That is, $n!=n(n-1)(n-2) \\cdots(3)(2)(1)$. For example, $4!=4(3)(2)(1)=24$ and $1!=1$. If $a$ and $b$ are positive integers with $b>a$, what is the ones (units) digit of $b!-a$! that cannot be?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 3174,
        "difficulty": 2.5
      },
      {
        "id": "4b6eef6121f6b83fc82df73a3a46d7048b1ff1b0c58dbf1f9efcd8e7ff5c83c7",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=7, B C=9$, and $C A=4$. Let $D$ be the point such that $A B \\| C D$ and $C A \\| B D$. Let $R$ be a point within triangle $B C D$. Lines $\\ell$ and $m$ going through $R$ are parallel to $C A$ and $A B$ respectively. Line $\\ell$ meets $A B$ and $B C$ at $P$ and $P^{\\prime}$ respectively, and $m$ meets $C A$ and $B C$ at $Q$ and $Q^{\\prime}$ respectively. If $S$ denotes the largest possible sum of the areas of triangles $B P P^{\\prime}, R P^{\\prime} Q^{\\prime}$, and $C Q Q^{\\prime}$, determine the value of $S^{2}$.",
        "answer": "180",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1637,
        "difficulty": 5.5
      },
      {
        "id": "3a2ca8955194a84fe62ad8b14bfaa38a633b4953ea83c072933d1f27a301af16",
        "domain": "omni",
        "question": "Let $q(x)=q^{1}(x)=2x^{2}+2x-1$, and let $q^{n}(x)=q(q^{n-1}(x))$ for $n>1$. How many negative real roots does $q^{2016}(x)$ have?",
        "answer": "\\frac{2017+1}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 945,
        "difficulty": 4.5
      },
      {
        "id": "7b321a8ed571e1b4ea491a74a867000f438a510e98f9e0ec88247ef5c5da7515",
        "domain": "omni",
        "question": "Suppose $E, I, L, V$ are (not necessarily distinct) nonzero digits in base ten for which the four-digit number $\\underline{E} \\underline{V} \\underline{I} \\underline{L}$ is divisible by 73 , and the four-digit number $\\underline{V} \\underline{I} \\underline{L} \\underline{E}$ is divisible by 74 . Compute the four-digit number $\\underline{L} \\underline{I} \\underline{V} \\underline{E}$.",
        "answer": "9954",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1221,
        "difficulty": 4.5
      },
      {
        "id": "951866a92743e3c6abafd1ccef18901c1fd71e9d80a3f56844dba65e3456ad90",
        "domain": "omni",
        "question": "Let $C$ be a circle with two diameters intersecting at an angle of 30 degrees. A circle $S$ is tangent to both diameters and to $C$, and has radius 1. Find the largest possible radius of $C$.",
        "answer": "1+\\sqrt{2}+\\sqrt{6}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 1087,
        "difficulty": 4.0
      },
      {
        "id": "c56c67cf721b6b46d41333f24d31aa4d6ffcd7a939339d058a03082e9461e33b",
        "domain": "omni",
        "question": "What is the result of subtracting eighty-seven from nine hundred forty-three?",
        "answer": "856",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2934,
        "difficulty": 1.0
      },
      {
        "id": "115b1cbccacf0d2f45f2cb5353fc148614cac3e69074817c9cbc789c9bb75232",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $C A=C B=5$ and $A B=8$. A circle $\\omega$ is drawn such that the interior of triangle $A B C$ is completely contained in the interior of $\\omega$. Find the smallest possible area of $\\omega$.",
        "answer": "16 \\pi",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1765,
        "difficulty": 4.0
      },
      {
        "id": "cda83f69ad03527095daabdda6f3df211c1b3a8e67523e6d79c6384df828736b",
        "domain": "omni",
        "question": "Jacob flipped a fair coin five times. In the first three flips, the coin came up heads exactly twice. In the last three flips, the coin also came up heads exactly twice. What is the probability that the third flip was heads?",
        "answer": "\\frac{4}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2472,
        "difficulty": 3.5
      },
      {
        "id": "6051b148bd0e8fc8997f8ebc296ab9f125c04e6a12619422370def41595a0ffb",
        "domain": "omni",
        "question": "Evaluate the infinite sum $\\sum_{n=0}^{\\infty}\\binom{2 n}{n} \\frac{1}{5^{n}}$.",
        "answer": "\\sqrt{5}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "HMMT_2",
        "source_index": 1398,
        "difficulty": 5.0
      },
      {
        "id": "880aecaaae5b18116fb565d4010dd697a2929636d820ec02c5bbc7621288479c",
        "domain": "omni",
        "question": "Let $a$ and $b$ be positive real numbers. Determine the minimum possible value of $$\\sqrt{a^{2}+b^{2}}+\\sqrt{(a-1)^{2}+b^{2}}+\\sqrt{a^{2}+(b-1)^{2}}+\\sqrt{(a-1)^{2}+(b-1)^{2}}$$",
        "answer": "2 \\sqrt{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1910,
        "difficulty": 5.0
      },
      {
        "id": "12b30af7dee8a9332b858c238fb2bc9d175b51364eb76591fdb936de5b176423",
        "domain": "omni",
        "question": "In how many ways can we fill the cells of a $4\\times4$ grid such that each cell contains exactly one positive integer and the product of the numbers in each row and each column is $2020$?",
        "answer": "576",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "lusophon_mathematical_olympiad",
        "source_index": 3999,
        "difficulty": 5.0
      },
      {
        "id": "20085184779b39510bbfba852e765a6ae87649313dd4ee7807a5af4979f2646c",
        "domain": "omni",
        "question": "A ten-level 2-tree is drawn in the plane: a vertex $A_{1}$ is marked, it is connected by segments with two vertices $B_{1}$ and $B_{2}$, each of $B_{1}$ and $B_{2}$ is connected by segments with two of the four vertices $C_{1}, C_{2}, C_{3}, C_{4}$ (each $C_{i}$ is connected with one $B_{j}$ exactly); and so on, up to 512 vertices $J_{1}, \\ldots, J_{512}$. Each of the vertices $J_{1}, \\ldots, J_{512}$ is coloured blue or golden. Consider all permutations $f$ of the vertices of this tree, such that (i) if $X$ and $Y$ are connected with a segment, then so are $f(X)$ and $f(Y)$, and (ii) if $X$ is coloured, then $f(X)$ has the same colour. Find the maximum $M$ such that there are at least $M$ permutations with these properties, regardless of the colouring.",
        "answer": "2^{2^{7}}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "izho",
        "source_index": 1855,
        "difficulty": 7.0
      },
      {
        "id": "af2a0bce5187a04a64b6c46e7c910a5ca7da314cc86e7a91ad84c5a4fcdf2156",
        "domain": "omni",
        "question": "In the decimal representation of $\frac{1}{7}$, the 100th digit to the right of the decimal is?",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Decimals",
        "source": "pascal",
        "source_index": 3508,
        "difficulty": 1.5
      },
      {
        "id": "b688d36e1b5dbd5e19dccd5882091e421ed6e8f34a71123264d3bed9ed7269cc",
        "domain": "omni",
        "question": "Yann writes down the first $n$ consecutive positive integers, $1,2,3,4, \\ldots, n-1, n$. He removes four different integers $p, q, r, s$ from the list. At least three of $p, q, r, s$ are consecutive and $100<p<q<r<s$. The average of the integers remaining in the list is 89.5625. What is the number of possible values of $s$?",
        "answer": "22",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3046,
        "difficulty": 1.5
      },
      {
        "id": "42e15f4d2bf6df07f9ddc095cd33a904bf3e5f2cc9105473c8310466f5f9fc3a",
        "domain": "omni",
        "question": "Count the number of sequences $a_{1}, a_{2}, a_{3}, a_{4}, a_{5}$ of integers such that $a_{i} \\leq 1$ for all $i$ and all partial sums $\\left(a_{1}, a_{1}+a_{2}\\right.$, etc.) are non-negative.",
        "answer": "132",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_2",
        "source_index": 3306,
        "difficulty": 4.5
      },
      {
        "id": "11a9238963179385cfdcb4a2a0eedbc1276cd9378ee4ee74b10e7e1585164812",
        "domain": "omni",
        "question": "A rectangular prism has a volume of $12 \\mathrm{~cm}^{3}$. A new prism is formed by doubling the length, doubling the width, and tripling the height of the original prism. What is the volume of this new prism?",
        "answer": "144",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume",
        "source": "pascal",
        "source_index": 3013,
        "difficulty": 1.5
      },
      {
        "id": "9da09096302a8165700a16e31cba1678764dacb36909ad68cf206350b6da8d97",
        "domain": "omni",
        "question": "Find the number of positive integers less than 1000000 which are less than or equal to the sum of their proper divisors. If your answer is $X$ and the actual value is $Y$, your score will be $\\max \\left(0,20-80\\left|1-\\frac{X}{Y}\\right|\\right)$ rounded to the nearest integer.",
        "answer": "247548",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1956,
        "difficulty": 4.0
      },
      {
        "id": "cb04dbad98304a310fe6653a0cc7e103c22f3ab62e907fd04de1f8a12d403488",
        "domain": "omni",
        "question": "Let $f(n)$ be the largest prime factor of $n$. Estimate $$N=\\left\\lfloor 10^{4} \\cdot \\frac{\\sum_{n=2}^{10^{6}} f\\left(n^{2}-1\\right)}{\\sum_{n=2}^{10^{6}} f(n)}\\right\\rfloor$$ An estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{3}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
        "answer": "18215",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1147,
        "difficulty": 5.25
      },
      {
        "id": "bd7fa2536ac8efd4a6fed56904291d308a314277655cdcd2ebf8990f29392cc4",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that $\\underbrace{2^{2 \\cdot 2}}_{n}>3^{3^{3^{3}}}$. (The notation $\\underbrace{2^{2^{2}}}_{n}$, is used to denote a power tower with $n 2$ 's. For example, $\\underbrace{2^{22^{2}}}_{n}$ with $n=4$ would equal $2^{2^{2^{2}}}$.)",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 2282,
        "difficulty": 4.0
      },
      {
        "id": "4aca5b81b224b2ed2549d7c2449be2c8fe2b64cbe75f5edd42d03bd5b2d51f65",
        "domain": "omni",
        "question": "For each positive integer $n$ let $S_{n}$ denote the set $\\{1,2,3, \\ldots, n\\}$. Compute the number of triples of subsets $A, B, C$ of $S_{2006}$ (not necessarily nonempty or proper) such that $A$ is a subset of $B$ and $S_{2006}-A$ is a subset of $C$.",
        "answer": "2^{4012}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 629,
        "difficulty": 5.25
      },
      {
        "id": "39d6c6e235f64516d61cb1ddc40a323d835854fd89d22e8d2d9bb777286afa66",
        "domain": "omni",
        "question": "The English alphabet, which has 26 letters, is randomly permuted. Let \\(p_{1}\\) be the probability that \\(\\mathrm{AB}, \\mathrm{CD}\\), and \\(\\mathrm{EF}\\) all appear as contiguous substrings. Let \\(p_{2}\\) be the probability that \\(\\mathrm{ABC}\\) and \\(\\mathrm{DEF}\\) both appear as contiguous substrings. Compute \\(\\frac{p_{1}}{p_{2}}\\).",
        "answer": "23",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2347,
        "difficulty": 5.0
      },
      {
        "id": "add431307634dd32ffe69b77684409f87516e3c9f5ecf7c9df32b798ceafea33",
        "domain": "omni",
        "question": "All the sequences consisting of five letters from the set $\\{T, U, R, N, I, P\\}$ (with repetitions allowed) are arranged in alphabetical order in a dictionary. Two sequences are called \"anagrams\" of each other if one can be obtained by rearranging the letters of the other. How many pairs of anagrams are there that have exactly 100 other sequences between them in the dictionary?",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 430,
        "difficulty": 4.0
      },
      {
        "id": "590338c254dbc4452304848923eb5f717cdf7e83412b88f1220ba1f0c64177e3",
        "domain": "omni",
        "question": "Sean is a biologist, and is looking at a string of length 66 composed of the letters $A, T, C, G$. A substring of a string is a contiguous sequence of letters in the string. For example, the string $AGTC$ has 10 substrings: $A, G, T, C, AG, GT, TC, AGT, GTC, AGTC$. What is the maximum number of distinct substrings of the string Sean is looking at?",
        "answer": "2100",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 488,
        "difficulty": 5.0
      },
      {
        "id": "87f99dbd47c8d80d55dc46ff11974d2c9fd7e9eee058b8f7d6225bc3900996e0",
        "domain": "omni",
        "question": "Let $A$ be the number of unordered pairs of ordered pairs of integers between 1 and 6 inclusive, and let $B$ be the number of ordered pairs of unordered pairs of integers between 1 and 6 inclusive. (Repetitions are allowed in both ordered and unordered pairs.) Find $A-B$.",
        "answer": "225",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1911,
        "difficulty": 5.0
      },
      {
        "id": "fc4f8ab6985e868d49e3a11ffed7ede2a1b26efdc70be20bea36fe2eade42a87",
        "domain": "omni",
        "question": "Kermit the frog enjoys hopping around the infinite square grid in his backyard. It takes him 1 Joule of energy to hop one step north or one step south, and 1 Joule of energy to hop one step east or one step west. He wakes up one morning on the grid with 100 Joules of energy, and hops till he falls asleep with 0 energy. How many different places could he have gone to sleep?",
        "answer": "10201",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1654,
        "difficulty": 4.0
      },
      {
        "id": "64739bdd840b6a92d5e84cffdc18291cc66effbbae557ef1bc23d386962ded67",
        "domain": "omni",
        "question": "Find the largest integer $n$ such that $n$ is divisible by all positive integers less than $\\sqrt[3]{n}$.",
        "answer": "420",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "apmo",
        "source_index": 3583,
        "difficulty": 6.0
      },
      {
        "id": "3aa1356cd2e33167746641aee2f922e14caadf0632ccd2861ac0bed7143fa9a7",
        "domain": "omni",
        "question": "Compute the prime factorization of 1007021035035021007001.",
        "answer": "7^{7} \\cdot 11^{7} \\cdot 13^{7}",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 784,
        "difficulty": 5.0
      },
      {
        "id": "9c35df5751b7bac9f95cd7528dc8fc5921ef41fa62a8a1799997c75e7118e32d",
        "domain": "omni",
        "question": "Find the total area of the region outside of an equilateral triangle but inside three circles each with radius 1, centered at the vertices of the triangle.",
        "answer": "\\frac{2 \\pi-\\sqrt{3}}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2531,
        "difficulty": 4.5
      },
      {
        "id": "0575448459ff7f152bb581700eb4bfb136a57ee728f2cfd6753fc0520ac28ef1",
        "domain": "omni",
        "question": "Determine the value of $$2002+\\frac{1}{2}\\left(2001+\\frac{1}{2}\\left(2000+\\cdots+\\frac{1}{2}\\left(3+\\frac{1}{2} \\cdot 2\\right)\\right) \\cdots\\right)$$",
        "answer": "4002",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1168,
        "difficulty": 2.5
      },
      {
        "id": "b4771ca6dc43ed16d14581e3c111aff4ad8efff06ed21ba26aacbad6d6523186",
        "domain": "omni",
        "question": "You are trying to cross a 400 foot wide river. You can jump at most 4 feet, but you have many stones you can throw into the river. You will stop throwing stones and cross the river once you have placed enough stones to be able to do so. You can throw straight, but you can't judge distance very well, so each stone ends up being placed uniformly at random along the width of the river. Estimate the expected number $N$ of stones you must throw before you can get across the river. An estimate of $E$ will earn $\\left\\lfloor 20 \\min \\left(\\frac{N}{E}, \\frac{E}{N}\\right)^{3}\\right\\rfloor$ points.",
        "answer": "712.811",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2025,
        "difficulty": 5.25
      },
      {
        "id": "24df1e2db6f282b8887bfa2d02abb37e5e0f1e63014fb5608113d2a2588852c2",
        "domain": "omni",
        "question": "A two-digit positive integer $x$ has the property that when 109 is divided by $x$, the remainder is 4. What is the sum of all such two-digit positive integers $x$?",
        "answer": "71",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "cayley",
        "source_index": 2684,
        "difficulty": 2.0
      },
      {
        "id": "6cc4cb77560842e7c7b0906547dec86f58aad3384a6bc71546508b578ccfee76",
        "domain": "omni",
        "question": "For odd primes $p$, let $f(p)$ denote the smallest positive integer $a$ for which there does not exist an integer $n$ satisfying $p \\mid n^{2}-a$. Estimate $N$, the sum of $f(p)^{2}$ over the first $10^{5}$ odd primes $p$. An estimate of $E>0$ will receive $\\left\\lfloor 22 \\min (N / E, E / N)^{3}\\right\\rfloor$ points.",
        "answer": "2266067",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 558,
        "difficulty": 5.25
      },
      {
        "id": "67d85a557ed55e4159d4bffe213e0a5bd11fe385c9c2165ac49db1e51eea7934",
        "domain": "omni",
        "question": "A committee of 5 is to be chosen from a group of 9 people. How many ways can it be chosen, if Bill and Karl must serve together or not at all, and Alice and Jane refuse to serve with each other?",
        "answer": "41",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1190,
        "difficulty": 4.0
      },
      {
        "id": "02bbd33f4b49ca9254f79bb09b1adcfe7de164b54615964a26768ec3ad199d73",
        "domain": "omni",
        "question": "Find the number of ways in which the letters in \"HMMTHMMT\" can be rearranged so that each letter is adjacent to another copy of the same letter. For example, \"MMMMTTHH\" satisfies this property, but \"HHTMMMTM\" does not.",
        "answer": "12",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2572,
        "difficulty": 4.0
      },
      {
        "id": "811a0083a1947f2e6a3107b1c69c3a9bcfbc88e59bbc10a0e904d07041d56b60",
        "domain": "omni",
        "question": "Let $N$ be the number of sequences of positive integers $\\left(a_{1}, a_{2}, a_{3}, \\ldots, a_{15}\\right)$ for which the polynomials $$x^{2}-a_{i} x+a_{i+1}$$ each have an integer root for every $1 \\leq i \\leq 15$, setting $a_{16}=a_{1}$. Estimate $N$. An estimate of $E$ will earn $\\left\\lfloor 20 \\min \\left(\\frac{N}{E}, \\frac{E}{N}\\right)^{2}\\right\\rfloor$ points.",
        "answer": "1409",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "HMMT_11",
        "source_index": 1916,
        "difficulty": 5.0
      },
      {
        "id": "a4e06e24e244e1a74c11476480cb7e9536a2f65911ae54ac80c6e0b70ebaf51f",
        "domain": "omni",
        "question": "Suppose that the plane is tiled with an infinite checkerboard of unit squares. If another unit square is dropped on the plane at random with position and orientation independent of the checkerboard tiling, what is the probability that it does not cover any of the corners of the squares of the checkerboard?",
        "answer": "2 - \\frac{6}{\\pi}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "putnam",
        "source_index": 3253,
        "difficulty": 7.0
      },
      {
        "id": "2f78c4588630c40fa65db5489d8ffbda9d356a109aac8990e5e489bc762005d8",
        "domain": "omni",
        "question": "Regular hexagon $A B C D E F$ has side length 2. A laser beam is fired inside the hexagon from point $A$ and hits $\\overline{B C}$ at point $G$. The laser then reflects off $\\overline{B C}$ and hits the midpoint of $\\overline{D E}$. Find $B G$.",
        "answer": "\\frac{2}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 1736,
        "difficulty": 4.0
      },
      {
        "id": "8b325d1f005a4ff495eaabbec7d242a2f30daa9f3dd17d9312f9366ef23502e8",
        "domain": "omni",
        "question": "Square $P Q R S$ has an area of 900. $M$ is the midpoint of $P Q$ and $N$ is the midpoint of $P S$. What is the area of triangle $P M N$?",
        "answer": "112.5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "cayley",
        "source_index": 2692,
        "difficulty": 1.5
      },
      {
        "id": "43672f450418c3fe38c8cfa85e0c7433c0d90dee1e39727b27bb3114928c3bd0",
        "domain": "omni",
        "question": "A five-digit positive integer is created using each of the odd digits $1, 3, 5, 7, 9$ once so that the thousands digit is larger than the hundreds digit, the thousands digit is larger than the ten thousands digit, the tens digit is larger than the hundreds digit, and the tens digit is larger than the units digit. How many such five-digit positive integers are there?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "cayley",
        "source_index": 3082,
        "difficulty": 2.5
      },
      {
        "id": "234c9f870fa6f0fd14143b78afb8179dd09e6f3bea3b3ac79e76b42af92c097b",
        "domain": "omni",
        "question": "Find the sum of the $x$-coordinates of the distinct points of intersection of the plane curves given by $x^{2}=x+y+4$ and $y^{2}=y-15 x+36$.",
        "answer": "0",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1332,
        "difficulty": 4.0
      },
      {
        "id": "5a8e46167195f6d9e06dab56f81e11c5a144c144aaaa6c8e9d7df07150109b80",
        "domain": "omni",
        "question": "Compute the number of positive integers less than 10! which can be expressed as the sum of at most 4 (not necessarily distinct) factorials.",
        "answer": "648",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2658,
        "difficulty": 5.0
      },
      {
        "id": "db6b5cdd43ea794f9d96b2b7701a30e949cd7f0fa0e1c9d2297073c81a6d1327",
        "domain": "omni",
        "question": "An omino is a 1-by-1 square or a 1-by-2 horizontal rectangle. An omino tiling of a region of the plane is a way of covering it (and only it) by ominoes. How many omino tilings are there of a 2-by-10 horizontal rectangle?",
        "answer": "7921",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 754,
        "difficulty": 4.5
      },
      {
        "id": "76311223ab2cd9a43c36c3f842a4538499e8b32596b4490e60550f90190bd7d7",
        "domain": "omni",
        "question": "Let $r_{k}$ denote the remainder when $\\binom{127}{k}$ is divided by 8. Compute $r_{1}+2 r_{2}+3 r_{3}+\\cdots+63 r_{63}$.",
        "answer": "8096",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1949,
        "difficulty": 7.0
      },
      {
        "id": "6a89794bddc96b5b57c2fe86530c3e57e9b1550cf041a9741d1ac31858382a3b",
        "domain": "omni",
        "question": "If $y=1$ and $4x-2y+3=3x+3y$, what is the value of $x$?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2894,
        "difficulty": 1.0
      },
      {
        "id": "c773b0dfdfc81e4bc0da7b0e98c78f772a59a76383e0dcf3311ca1034abe0395",
        "domain": "omni",
        "question": "To survive the coming Cambridge winter, Chim Tu doesn't wear one T-shirt, but instead wears up to FOUR T-shirts, all in different colors. An outfit consists of three or more T-shirts, put on one on top of the other in some order, such that two outfits are distinct if the sets of T-shirts used are different or the sets of T-shirts used are the same but the order in which they are worn is different. Given that Chim Tu changes his outfit every three days, and otherwise never wears the same outfit twice, how many days of winter can Chim Tu survive? (Needless to say, he only has four t-shirts.)",
        "answer": "144",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Permutations",
        "source": "HMMT_11",
        "source_index": 1935,
        "difficulty": 4.0
      },
      {
        "id": "ed60d15961d91fd973a43f000dff46140f58b68628cd752bc4fe5f496b7189c1",
        "domain": "omni",
        "question": "If \\( 3^x = 5 \\), what is the value of \\( 3^{x+2} \\)?",
        "answer": "45",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "cayley",
        "source_index": 3019,
        "difficulty": 2.5
      },
      {
        "id": "1715f01d5e8a45873052579c3ac35a754fb201829000971901e6fe74e3b0b574",
        "domain": "omni",
        "question": "Let $\\alpha, \\beta$, and $\\gamma$ be three real numbers. Suppose that $\\cos \\alpha+\\cos \\beta+\\cos \\gamma =1$ and $\\sin \\alpha+\\sin \\beta+\\sin \\gamma =1$. Find the smallest possible value of $\\cos \\alpha$.",
        "answer": "\\frac{-1-\\sqrt{7}}{4}",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions; Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1589,
        "difficulty": 5.0
      },
      {
        "id": "2d17f903e53b3fe477e598032cb0c9ab6058c8fe155a9b2177c96b82b8710e61",
        "domain": "omni",
        "question": "Paul and Sara are playing a game with integers on a whiteboard, with Paul going first. When it is Paul's turn, he can pick any two integers on the board and replace them with their product; when it is Sara's turn, she can pick any two integers on the board and replace them with their sum. Play continues until exactly one integer remains on the board. Paul wins if that integer is odd, and Sara wins if it is even. Initially, there are 2021 integers on the board, each one sampled uniformly at random from the set \\{0,1,2,3, \\ldots, 2021\\}. Assuming both players play optimally, the probability that Paul wins is $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find the remainder when $m+n$ is divided by 1000.",
        "answer": "\\[\n383\n\\]",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 463,
        "difficulty": 4.5
      },
      {
        "id": "a4978088afa95cdb697a2b43d2e75823b757ae12c01430d183e263286daefc56",
        "domain": "omni",
        "question": "David and Evan are playing a game. Evan thinks of a positive integer $N$ between 1 and 59, inclusive, and David tries to guess it. Each time David makes a guess, Evan will tell him whether the guess is greater than, equal to, or less than $N$. David wants to devise a strategy that will guarantee that he knows $N$ in five guesses. In David's strategy, each guess will be determined only by Evan's responses to any previous guesses (the first guess will always be the same), and David will only guess a number which satisfies each of Evan's responses. How many such strategies are there?",
        "answer": "36440",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 1898,
        "difficulty": 4.5
      },
      {
        "id": "73876e2f5f0162701952f0fb13846fd739ea44caca6811072122cd3bcad3038a",
        "domain": "omni",
        "question": "An unfair coin has the property that when flipped four times, it has the same probability of turning up 2 heads and 2 tails (in any order) as 3 heads and 1 tail (in any order). What is the probability of getting a head in any one flip?",
        "answer": "\\frac{3}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1126,
        "difficulty": 4.0
      },
      {
        "id": "6134b6fcbd7490e0dce63df3bd09cee4eabcb266f459d25d0ad45f9d6a83028d",
        "domain": "omni",
        "question": "A plane $P$ slices through a cube of volume 1 with a cross-section in the shape of a regular hexagon. This cube also has an inscribed sphere, whose intersection with $P$ is a circle. What is the area of the region inside the regular hexagon but outside the circle?",
        "answer": "(3 \\sqrt{3}-\\pi) / 4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 732,
        "difficulty": 4.0
      },
      {
        "id": "34e91a4c3dfb3a1d2d73002bf39246909859f195a30b24d9fdd91f0f973e4fe1",
        "domain": "omni",
        "question": "In how many ways can 6 purple balls and 6 green balls be placed into a $4 \\times 4$ grid of boxes such that every row and column contains two balls of one color and one ball of the other color? Only one ball may be placed in each box, and rotations and reflections of a single configuration are considered different.",
        "answer": "5184",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 999,
        "difficulty": 5.0
      },
      {
        "id": "d7e517c188648ba3a0fe2234a8273e259b7fc0e269c940d6649b4116bca9cbba",
        "domain": "omni",
        "question": "What is the value of $2^{4}-2^{3}$?",
        "answer": "2^{3}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2850,
        "difficulty": 1.5
      },
      {
        "id": "ac06aebea1fb7899e2198679cc9c5fc88ec6c57d79bd6363ca4fc9cac930fae4",
        "domain": "omni",
        "question": "The sum of four different positive integers is 100. The largest of these four integers is $n$. What is the smallest possible value of $n$?",
        "answer": "27",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2754,
        "difficulty": 2.5
      },
      {
        "id": "cd19abfc3de49b69e087dd3121b9d3159b3671e5f9829469e9064d80e024d5f6",
        "domain": "omni",
        "question": "Triangle $A B C$ has $A B=1, B C=\\sqrt{7}$, and $C A=\\sqrt{3}$. Let $\\ell_{1}$ be the line through $A$ perpendicular to $A B, \\ell_{2}$ the line through $B$ perpendicular to $A C$, and $P$ the point of intersection of $\\ell_{1}$ and $\\ell_{2}$. Find $P C$.",
        "answer": "3",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_2",
        "source_index": 967,
        "difficulty": 4.5
      },
      {
        "id": "fda61937b30bdf6e90eccf6969200624f943bfe76e1bc32b3a5a479e8f16846e",
        "domain": "omni",
        "question": "Which of the following integers is equal to a perfect square: $2^{3}$, $3^{5}$, $4^{7}$, $5^{9}$, $6^{11}$?",
        "answer": "4^{7}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Prime Numbers",
        "source": "cayley",
        "source_index": 3422,
        "difficulty": 2.0
      },
      {
        "id": "4623ec578782bc6ae14e5eabf35895fc46d7d2aaed797d325a3099672ecabf25",
        "domain": "omni",
        "question": "Call the pentominoes found in the last problem square pentominoes. Just like dominos and ominos can be used to tile regions of the plane, so can square pentominoes. In particular, a square pentomino tiling of a region of the plane is a way of covering it (and only it) completely by nonoverlapping square pentominoes. How many square pentomino tilings are there of a 12-by-12 rectangle?",
        "answer": "0",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 782,
        "difficulty": 5.0
      },
      {
        "id": "72c6211559b06f663ca7c94d131004ff55b03753d58e935a8cd9ca2aa6274691",
        "domain": "omni",
        "question": "Anna walked at a constant rate. If she walked 600 metres in 4 minutes, how far did she walk in 6 minutes?",
        "answer": "900",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 3449,
        "difficulty": 1.0
      },
      {
        "id": "6f477959e109458fea170c74613b5248688494c45de0fde4e2112968488e7b14",
        "domain": "omni",
        "question": "What is the smallest possible perimeter of a triangle whose side lengths are all squares of distinct positive integers?",
        "answer": "77",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2024,
        "difficulty": 4.5
      },
      {
        "id": "9a2374574e3318129499e5400d0c90e57916b47ba4aa369d3f6f7f7514314c86",
        "domain": "omni",
        "question": "Triangle $A B C$ has side lengths $A B=15, B C=18, C A=20$. Extend $C A$ and $C B$ to points $D$ and $E$ respectively such that $D A=A B=B E$. Line $A B$ intersects the circumcircle of $C D E$ at $P$ and $Q$. Find the length of $P Q$.",
        "answer": "37",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2588,
        "difficulty": 5.25
      },
      {
        "id": "6e0852dfe45cabd936f2e44a370986c6bd64e5532803ca62885c0264a5abf50a",
        "domain": "omni",
        "question": "Rachelle picks a positive integer \\(a\\) and writes it next to itself to obtain a new positive integer \\(b\\). For instance, if \\(a=17\\), then \\(b=1717\\). To her surprise, she finds that \\(b\\) is a multiple of \\(a^{2}\\). Find the product of all the possible values of \\(\\frac{b}{a^{2}}\\).",
        "answer": "77",
        "topic": "Mathematics -> Algebra -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 333,
        "difficulty": 5.0
      },
      {
        "id": "25607717bd05edf664e50bf6f941c359db59bcc33c0db60b977dc1b832b240eb",
        "domain": "omni",
        "question": "Convex quadrilateral $B C D E$ lies in the plane. Lines $E B$ and $D C$ intersect at $A$, with $A B=2$, $A C=5, A D=200, A E=500$, and $\\cos \\angle B A C=\\frac{7}{9}$. What is the largest number of nonoverlapping circles that can lie in quadrilateral $B C D E$ such that all of them are tangent to both lines $B E$ and $C D$ ?",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2590,
        "difficulty": 5.0
      },
      {
        "id": "5b575380b93a72d04e420ead80481bf80fa964d03c8edd2fe6eb9093cc8879ee",
        "domain": "omni",
        "question": "What fraction of the entire wall is painted red if Matilda paints half of her section red and Ellie paints one third of her section red?",
        "answer": "\\frac{5}{12}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3404,
        "difficulty": 1.5
      },
      {
        "id": "59eb610d97a1bd183cea5728bb51cb012b9968f2dff1283920b11ffb76e3ea3e",
        "domain": "omni",
        "question": "A subset of a student group is called an [i]ideal company[/i] if\n1) in this subset, all girls are liked by all young men,\n2) no one can be added to this subset without violating condition $1$.\nIn a certain group, $9$ female students and $15$ students study. Warden of the group made a list of all kinds of ideal companies in this group. What is the largest number of companies on this list?",
        "answer": "512",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Logic",
        "source": "ToT",
        "source_index": 4039,
        "difficulty": 5.5
      },
      {
        "id": "2216e498828b681bb057e74d49497f7f348ec28493e43cd233cc4d6b60eaefc7",
        "domain": "omni",
        "question": "There are six empty slots corresponding to the digits of a six-digit number. Claire and William take turns rolling a standard six-sided die, with Claire going first. They alternate with each roll until they have each rolled three times. After a player rolls, they place the number from their die roll into a remaining empty slot of their choice. Claire wins if the resulting six-digit number is divisible by 6, and William wins otherwise. If both players play optimally, compute the probability that Claire wins.",
        "answer": "\\frac{43}{192}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1922,
        "difficulty": 4.5
      },
      {
        "id": "e7b2a8474312ac6517a7c0dce17f180ca31dd305751b15339e5640f9e185b8b3",
        "domain": "omni",
        "question": "Let $a, b$ be integers chosen independently and uniformly at random from the set $\\{0,1,2, \\ldots, 80\\}$. Compute the expected value of the remainder when the binomial coefficient $\\binom{a}{b}=\\frac{a!}{b!(a-b)!}$ is divided by 3.",
        "answer": "\\frac{1816}{6561}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 745,
        "difficulty": 5.0
      },
      {
        "id": "65cf19266655e48f7de3bb68dfe5522997590bf98dddc96ee8d52d363e63f7f5",
        "domain": "omni",
        "question": "For how many positive integers $k$ do the lines with equations $9x+4y=600$ and $kx-4y=24$ intersect at a point whose coordinates are positive integers?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2886,
        "difficulty": 2.5
      },
      {
        "id": "a875b9221cbd800f00c79bf1bdb321feaa3c8dd28b760497240e0a63c69043ed",
        "domain": "omni",
        "question": "We are given some similar triangles. Their areas are $1^{2}, 3^{2}, 5^{2} \\ldots$, and $49^{2}$. If the smallest triangle has a perimeter of 4, what is the sum of all the triangles' perimeters?",
        "answer": "2500",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1644,
        "difficulty": 4.0
      },
      {
        "id": "ffcdc06a214c5097099f7683fc0c8acf020fe29fcbb361de63c16a35136c620f",
        "domain": "omni",
        "question": "Given complex number $z$, define sequence $z_{0}, z_{1}, z_{2}, \\ldots$ as $z_{0}=z$ and $z_{n+1}=2 z_{n}^{2}+2 z_{n}$ for $n \\geq 0$. Given that $z_{10}=2017$, find the minimum possible value of $|z|$.",
        "answer": "\\frac{\\sqrt[1024]{4035}-1}{2}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Algebra -> Sequences and Series -> Other",
        "source": "HMMT_11",
        "source_index": 2609,
        "difficulty": 5.0
      },
      {
        "id": "ccc750a1eecb26ef6f51351fedbcf1baedb378ca759c632aa18426508e3f56bd",
        "domain": "omni",
        "question": "Katie has a fair 2019-sided die with sides labeled $1,2, \\ldots, 2019$. After each roll, she replaces her $n$-sided die with an $(n+1)$-sided die having the $n$ sides of her previous die and an additional side with the number she just rolled. What is the probability that Katie's $2019^{\\text {th }}$ roll is a 2019?",
        "answer": "\\frac{1}{2019}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2174,
        "difficulty": 4.0
      },
      {
        "id": "ad1e59f8d39a69353ed84f37edb93ec6877536b2996fbb09c2ab0cd3d5af2778",
        "domain": "omni",
        "question": "An equiangular hexagon has side lengths $1,1, a, 1,1, a$ in that order. Given that there exists a circle that intersects the hexagon at 12 distinct points, we have $M<a<N$ for some real numbers $M$ and $N$. Determine the minimum possible value of the ratio $\\frac{N}{M}$.",
        "answer": "\\frac{3 \\sqrt{3}+3}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1947,
        "difficulty": 4.5
      },
      {
        "id": "279955c574c63e79aadc04373dda9492ea31c74bb03ecb8d741aaa6d121ec323",
        "domain": "omni",
        "question": "Kelvin the Frog was bored in math class one day, so he wrote all ordered triples $(a, b, c)$ of positive integers such that $a b c=2310$ on a sheet of paper. Find the sum of all the integers he wrote down. In other words, compute $$\\sum_{\\substack{a b c=2310 \\\\ a, b, c \\in \\mathbb{N}}}(a+b+c)$$ where $\\mathbb{N}$ denotes the positive integers.",
        "answer": "49140",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1382,
        "difficulty": 6.0
      },
      {
        "id": "81ff620766c97adf6d070300aa8b2960f7e5cef5e58fc2269102de87e8591f13",
        "domain": "omni",
        "question": "Suppose that $m$ and $n$ are positive integers with $\\sqrt{7+\\sqrt{48}}=m+\\sqrt{n}$. What is the value of $m^{2}+n^{2}$?",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "fermat",
        "source_index": 2762,
        "difficulty": 2.5
      },
      {
        "id": "d5e35a64c2c005e093507e8f571025d36ecf4aa016016b2a5a0a91298a78789a",
        "domain": "omni",
        "question": "Estimate $A$, the number of times an 8-digit number appears in Pascal's triangle. An estimate of $E$ earns $\\max (0,\\lfloor 20-|A-E| / 200\\rfloor)$ points.",
        "answer": "180020660",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2514,
        "difficulty": 5.25
      },
      {
        "id": "cfdff0d12d76450df1e8bac23fb3e2d9b4d94be6d2cfebb33f695182b03baa36",
        "domain": "omni",
        "question": "If $x$ and $y$ are positive integers with $xy = 6$, what is the sum of all possible values of $\\frac{2^{x+y}}{2^{x-y}}$?",
        "answer": "4180",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "cayley",
        "source_index": 2664,
        "difficulty": 2.0
      },
      {
        "id": "ae06dfc76d9c644065a86bf34c8c9016606054f88305dece481443bb3683b88a",
        "domain": "omni",
        "question": "The polynomial $f(x)=x^{3}-3 x^{2}-4 x+4$ has three real roots $r_{1}, r_{2}$, and $r_{3}$. Let $g(x)=x^{3}+a x^{2}+b x+c$ be the polynomial which has roots $s_{1}, s_{2}$, and $s_{3}$, where $s_{1}=r_{1}+r_{2} z+r_{3} z^{2}$, $s_{2}=r_{1} z+r_{2} z^{2}+r_{3}, s_{3}=r_{1} z^{2}+r_{2}+r_{3} z$, and $z=\\frac{-1+i \\sqrt{3}}{2}$. Find the real part of the sum of the coefficients of $g(x)$.",
        "answer": "-26",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 625,
        "difficulty": 5.5
      },
      {
        "id": "8625425646435708db5800c33e904c64606a0e38d63faa0b150f8d37b3cc53c9",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ for which $$1!2!\\cdots(n-1)!>n!^{2}$$",
        "answer": "8",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_11",
        "source_index": 1884,
        "difficulty": 4.5
      },
      {
        "id": "4dfb18f4fd3c804614d7a9d9e02f204ef004e3375164ffb1f358506456ebee9f",
        "domain": "omni",
        "question": "Four points are independently chosen uniformly at random from the interior of a regular dodecahedron. What is the probability that they form a tetrahedron whose interior contains the dodecahedron's center?",
        "answer": "\\[\n\\frac{1}{8}\n\\]",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 420,
        "difficulty": 5.25
      },
      {
        "id": "a3e525c228b25cf0e8e1015cb73a19ab492aa68ec77bb6308c8df24cc746c8a5",
        "domain": "omni",
        "question": "Compute the smallest positive integer $n$ for which $\\sqrt{100+\\sqrt{n}}+\\sqrt{100-\\sqrt{n}}$ is an integer.",
        "answer": "6156",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2148,
        "difficulty": 4.0
      },
      {
        "id": "e53231dacae927d97149fc5db41dc9f4f608cfce3611e38d5d64a04a1a89cfd7",
        "domain": "omni",
        "question": "Alberto, Bernardo, and Carlos are collectively listening to three different songs. Each is simultaneously listening to exactly two songs, and each song is being listened to by exactly two people. In how many ways can this occur?",
        "answer": "6",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2571,
        "difficulty": 3.0
      },
      {
        "id": "8f424d429cfb7c1c7fb32f6e7c5469441f65cfa81a01021468effcd18fe56335",
        "domain": "omni",
        "question": "The number $27,000,001$ has exactly four prime factors. Find their sum.",
        "answer": "652",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1286,
        "difficulty": 4.5
      },
      {
        "id": "5cd366c0d9d85da61e7c44514ffa375e898ac1aada698b4d3cc01b9f98b69d19",
        "domain": "omni",
        "question": "A cafe has 3 tables and 5 individual counter seats. People enter in groups of size between 1 and 4, inclusive, and groups never share a table. A group of more than 1 will always try to sit at a table, but will sit in counter seats if no tables are available. Conversely, a group of 1 will always try to sit at the counter first. One morning, $M$ groups consisting of a total of $N$ people enter and sit down. Then, a single person walks in, and realizes that all the tables and counter seats are occupied by some person or group. What is the minimum possible value of $M+N$?",
        "answer": "16",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1346,
        "difficulty": 4.5
      },
      {
        "id": "972a7a1b372b334c4c5231d337af66266e5a9564412d0596aa8404128c03b9e5",
        "domain": "omni",
        "question": "Connie has a number of gold bars, all of different weights. She gives the 24 lightest bars, which weigh $45 \\%$ of the total weight, to Brennan. She gives the 13 heaviest bars, which weigh $26 \\%$ of the total weight, to Maya. How many bars did Blair receive?",
        "answer": "15",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 2683,
        "difficulty": 2.0
      },
      {
        "id": "71c95b8d7185fae86a0ca1915fd7496cef09b59ec89dc69a0167a7e2f5e228e4",
        "domain": "omni",
        "question": "A semicircle with radius 2021 has diameter $AB$ and center $O$. Points $C$ and $D$ lie on the semicircle such that $\\angle AOC < \\angle AOD = 90^{\\circ}$. A circle of radius $r$ is inscribed in the sector bounded by $OA$ and $OC$ and is tangent to the semicircle at $E$. If $CD=CE$, compute $\\lfloor r \\rfloor$.",
        "answer": "673",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 942,
        "difficulty": 5.0
      },
      {
        "id": "a5db55a3d8d7580da403ca71f01a9da59c9d08e1779e44dfd70f0339e4e8e348",
        "domain": "omni",
        "question": "Let $S$ be a set of consecutive positive integers such that for any integer $n$ in $S$, the sum of the digits of $n$ is not a multiple of 11. Determine the largest possible number of elements of $S$.",
        "answer": "38",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1968,
        "difficulty": 5.0
      },
      {
        "id": "defed24d16622456a2eae7da9e02624c8ee7dc5bd7f1ec44a33865058ed867f2",
        "domain": "omni",
        "question": "In convex quadrilateral $ ABCD$, $ AB\\equal{}a$, $ BC\\equal{}b$, $ CD\\equal{}c$, $ DA\\equal{}d$, $ AC\\equal{}e$, $ BD\\equal{}f$. If $ \\max \\{a,b,c,d,e,f \\}\\equal{}1$, then find the maximum value of $ abcd$.",
        "answer": "2 - \\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "china_team_selection_test",
        "source_index": 144,
        "difficulty": 8.0
      },
      {
        "id": "5ca8a1c2e1602033f0dd73122b554166cd36947b25ccdf848007fb9224d904eb",
        "domain": "omni",
        "question": "Let $X$ be the collection of all functions $f:\\{0,1, \\ldots, 2016\\} \\rightarrow\\{0,1, \\ldots, 2016\\}$. Compute the number of functions $f \\in X$ such that $$\\max _{g \\in X}\\left(\\min _{0 \\leq i \\leq 2016}(\\max (f(i), g(i)))-\\max _{0 \\leq i \\leq 2016}(\\min (f(i), g(i)))\\right)=2015$$",
        "answer": "2 \\cdot\\left(3^{2017}-2^{2017}\\right)",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1200,
        "difficulty": 4.0
      },
      {
        "id": "6711233e266c21469370f20de7e0477d5d451306f0c59be91066594c237e7c58",
        "domain": "omni",
        "question": "Milan has a bag of 2020 red balls and 2021 green balls. He repeatedly draws 2 balls out of the bag uniformly at random. If they are the same color, he changes them both to the opposite color and returns them to the bag. If they are different colors, he discards them. Eventually the bag has 1 ball left. Let $p$ be the probability that it is green. Compute $\\lfloor 2021 p \\rfloor$.",
        "answer": "2021",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 833,
        "difficulty": 5.0
      },
      {
        "id": "3d7e3f671eebf3c38f6de8a74dccc23d914fa5b09e69de934e3a0e8107f56bff",
        "domain": "omni",
        "question": "As shown below, there is a $40\\times30$ paper with a filled $10\\times5$ rectangle inside of it. We want to cut out the filled rectangle from the paper using four straight cuts. Each straight cut is a straight line that divides the paper into two pieces, and we keep the piece containing the filled rectangle. The goal is to minimize the total length of the straight cuts. How to achieve this goal, and what is that minimized length? Show the correct cuts and write the final answer. There is no need to prove the answer.",
        "answer": "65",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "th_igo",
        "source_index": 3770,
        "difficulty": 5.0
      },
      {
        "id": "4086ba7d9137bda0d707cbe47f682fda2e26ad62cf5dcdfae7ffffac46ae592c",
        "domain": "omni",
        "question": "Lucy starts by writing $s$ integer-valued $2022$-tuples on a blackboard. After doing that, she can take any two (not necessarily distinct) tuples $\\mathbf{v}=(v_1,\\ldots,v_{2022})$ and $\\mathbf{w}=(w_1,\\ldots,w_{2022})$ that she has already written, and apply one of the following operations to obtain a new tuple:\n\\begin{align*}\n\\mathbf{v}+\\mathbf{w}&=(v_1+w_1,\\ldots,v_{2022}+w_{2022}) \\\\\n\\mathbf{v} \\lor \\mathbf{w}&=(\\max(v_1,w_1),\\ldots,\\max(v_{2022},w_{2022}))\n\\end{align*}\nand then write this tuple on the blackboard.\n\nIt turns out that, in this way, Lucy can write any integer-valued $2022$-tuple on the blackboard after finitely many steps. What is the smallest possible number $s$ of tuples that she initially wrote?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 3853,
        "difficulty": 9.0
      },
      {
        "id": "3119f527927dd4a096f1b922b8f8c2208da7eb19209f9edc807fa59b9b1ecb13",
        "domain": "omni",
        "question": "Elisenda has a piece of paper in the shape of a triangle with vertices $A, B$, and $C$ such that $A B=42$. She chooses a point $D$ on segment $A C$, and she folds the paper along line $B D$ so that $A$ lands at a point $E$ on segment $B C$. Then, she folds the paper along line $D E$. When she does this, $B$ lands at the midpoint of segment $D C$. Compute the perimeter of the original unfolded triangle.",
        "answer": "168+48 \\sqrt{7}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1517,
        "difficulty": 5.25
      },
      {
        "id": "d2bea71f772a33adc65a8ee5f7a1bc8e6a9d22d2750a52839c17fd6fa356e9a0",
        "domain": "omni",
        "question": "Let \\mathbb{N} denote the natural numbers. Compute the number of functions $f: \\mathbb{N} \\rightarrow\\{0,1, \\ldots, 16\\}$ such that $$f(x+17)=f(x) \\quad \\text { and } \\quad f\\left(x^{2}\\right) \\equiv f(x)^{2}+15 \\quad(\\bmod 17)$$ for all integers $x \\geq 1$",
        "answer": "12066",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1484,
        "difficulty": 6.0
      },
      {
        "id": "9c9793bfa104563a9670627cd2d55c588b91d967e8229d17e81a37431b6d3b3a",
        "domain": "omni",
        "question": "What is the probability that in a randomly chosen arrangement of the numbers and letters in \"HMMT2005,\" one can read either \"HMMT\" or \"2005\" from left to right?",
        "answer": "23/144",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1134,
        "difficulty": 3.5
      },
      {
        "id": "08a306011d94348e7ac7269c143940cfc951f8eee13ec0271353db7ed8c8ad94",
        "domain": "omni",
        "question": "Find the shortest distance from the line $3 x+4 y=25$ to the circle $x^{2}+y^{2}=6 x-8 y$.",
        "answer": "7 / 5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Distance -> Other",
        "source": "HMMT_2",
        "source_index": 640,
        "difficulty": 4.5
      },
      {
        "id": "061144ede36755042809a11840aac3c814cc7b75824127bccd7fa4c99a900bf4",
        "domain": "omni",
        "question": "Compute the number of ways to select 99 cells of a $19 \\times 19$ square grid such that no two selected cells share an edge or vertex.",
        "answer": "1000",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1600,
        "difficulty": 5.25
      },
      {
        "id": "1f56c5a20de5ec6f8f84dc843908e5c99de6210ec8182d588659201d6ffc0faa",
        "domain": "omni",
        "question": "Niffy's favorite number is a positive integer, and Stebbysaurus is trying to guess what it is. Niffy tells her that when expressed in decimal without any leading zeros, her favorite number satisfies the following: - Adding 1 to the number results in an integer divisible by 210 . - The sum of the digits of the number is twice its number of digits. - The number has no more than 12 digits. - The number alternates in even and odd digits. Given this information, what are all possible values of Niffy's favorite number?",
        "answer": "1010309",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 733,
        "difficulty": 5.0
      },
      {
        "id": "c9e3516d7af76b907452faeaba7593ebf198bc05fd15fd0e7ce26791726b93df",
        "domain": "omni",
        "question": "Let $X$ be the number of sequences of integers $a_{1}, a_{2}, \\ldots, a_{2047}$ that satisfy all of the following properties: - Each $a_{i}$ is either 0 or a power of 2 . - $a_{i}=a_{2 i}+a_{2 i+1}$ for $1 \\leq i \\leq 1023$ - $a_{1}=1024$. Find the remainder when $X$ is divided by 100 .",
        "answer": "15",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2518,
        "difficulty": 5.25
      },
      {
        "id": "7ea0bebf403f83bea24303637d22f545242bea7a5583e65871c7da05bad1f83b",
        "domain": "omni",
        "question": "We have 10 points on a line A_{1}, A_{2} \\cdots A_{10} in that order. Initially there are n chips on point A_{1}. Now we are allowed to perform two types of moves. Take two chips on A_{i}, remove them and place one chip on A_{i+1}, or take two chips on A_{i+1}, remove them, and place a chip on A_{i+2} and A_{i}. Find the minimum possible value of n such that it is possible to get a chip on A_{10} through a sequence of moves.",
        "answer": "46",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2110,
        "difficulty": 6.0
      },
      {
        "id": "e33e88d13d3c429f25294a099dae3f4f8b1abcd486c80cd876211b6880e1ef49",
        "domain": "omni",
        "question": "There is a $6 \\times 6$ grid of lights. There is a switch at the top of each column and on the left of each row. A light will only turn on if the switches corresponding to both its column and its row are in the \"on\" position. Compute the number of different configurations of lights.",
        "answer": "3970",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1804,
        "difficulty": 3.5
      },
      {
        "id": "698a39b77925d6736d8167569a7ea9199bf81c5d23c30ae0d1b807bfbf73b865",
        "domain": "omni",
        "question": "Let $f(n)$ be the number of distinct prime divisors of $n$ less than 6. Compute $$\\sum_{n=1}^{2020} f(n)^{2}$$",
        "answer": "3431",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2532,
        "difficulty": 5.0
      },
      {
        "id": "18c4374bf18bc6728699f3618805267e93926208c5c01b55b67c4262cb48ca3b",
        "domain": "omni",
        "question": "Two unit squares $S_{1}$ and $S_{2}$ have horizontal and vertical sides. Let $x$ be the minimum distance between a point in $S_{1}$ and a point in $S_{2}$, and let $y$ be the maximum distance between a point in $S_{1}$ and a point in $S_{2}$. Given that $x=5$, the difference between the maximum and minimum possible values for $y$ can be written as $a+b \\sqrt{c}$, where $a, b$, and $c$ are integers and $c$ is positive and square-free. Find $100 a+10 b+c$.",
        "answer": "472",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Distance Problems -> Other",
        "source": "HMMT_11",
        "source_index": 2594,
        "difficulty": 5.0
      },
      {
        "id": "287390818bd9c56b7e3ae9cf15d1d049015ebdf5dcf007ce32c0008f47daf081",
        "domain": "omni",
        "question": "Marcus and four of his relatives are at a party. Each pair of the five people are either friends or enemies. For any two enemies, there is no person that they are both friends with. In how many ways is this possible?",
        "answer": "52",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2332,
        "difficulty": 5.0
      },
      {
        "id": "a5efc8100a40600d67bf26a9ad4aea44b05817fd7f4613a2a324dd72cbccdca0",
        "domain": "omni",
        "question": "Find the remainder when $1^{2}+3^{2}+5^{2}+\\cdots+99^{2}$ is divided by 1000.",
        "answer": "650",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2570,
        "difficulty": 5.0
      },
      {
        "id": "3054ff6112a7fffa17db775565c6d1a4aa21cfe9182a24ae04ec0af2859f7189",
        "domain": "omni",
        "question": "New this year at HMNT: the exciting game of $R N G$ baseball! In RNG baseball, a team of infinitely many people play on a square field, with a base at each vertex; in particular, one of the bases is called the home base. Every turn, a new player stands at home base and chooses a number $n$ uniformly at random from \\{0,1,2,3,4\\}. Then, the following occurs: - If $n>0$, then the player and everyone else currently on the field moves (counterclockwise) around the square by $n$ bases. However, if in doing so a player returns to or moves past the home base, he/she leaves the field immediately and the team scores one point. - If $n=0$ (a strikeout), then the game ends immediately; the team does not score any more points. What is the expected number of points that a given team will score in this game?",
        "answer": "\\frac{409}{125}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2068,
        "difficulty": 4.0
      },
      {
        "id": "7780aaa0a0b99c2b3520f672b64fb3c9d60597773aa4c6fa059b6bb48bf62043",
        "domain": "omni",
        "question": "Find all real numbers $x$ such that $$x^{2}+\\left\\lfloor\\frac{x}{2}\\right\\rfloor+\\left\\lfloor\\frac{x}{3}\\right\\rfloor=10$$",
        "answer": "-\\sqrt{14}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 623,
        "difficulty": 3.5
      },
      {
        "id": "44ed80ecce8c9087e3f87441a055b8dd01828209325ef13a66779927f29186a8",
        "domain": "omni",
        "question": "Consider the function $z(x, y)$ describing the paraboloid $z=(2 x-y)^{2}-2 y^{2}-3 y$. Archimedes and Brahmagupta are playing a game. Archimedes first chooses $x$. Afterwards, Brahmagupta chooses $y$. Archimedes wishes to minimize $z$ while Brahmagupta wishes to maximize $z$. Assuming that Brahmagupta will play optimally, what value of $x$ should Archimedes choose?",
        "answer": "-\\frac{3}{8}",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives; Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "HMMT_2",
        "source_index": 3318,
        "difficulty": 5.5
      },
      {
        "id": "35584ad6b442b40a4e7c1721b82a1ff7f38b7902328009b0213662b48b89836e",
        "domain": "omni",
        "question": "A bag contains nine blue marbles, ten ugly marbles, and one special marble. Ryan picks marbles randomly from this bag with replacement until he draws the special marble. He notices that none of the marbles he drew were ugly. Given this information, what is the expected value of the number of total marbles he drew?",
        "answer": "\\frac{20}{11}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1107,
        "difficulty": 4.0
      },
      {
        "id": "e380d78a746bab1d127daf8e11a1567d10c9b7a361f84da2be45d03f2458f703",
        "domain": "omni",
        "question": "At the beginning of the first day, a box contains 1 black ball, 1 gold ball, and no other balls. At the end of each day, for each gold ball in the box, 2 black balls and 1 gold ball are added to the box. If no balls are removed from the box, how many balls are in the box at the end of the seventh day?",
        "answer": "383",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 2844,
        "difficulty": 2.0
      },
      {
        "id": "e39b6b9558a95e8c9d68acd84a5203957607118d148b482ea8466945e9b6af40",
        "domain": "omni",
        "question": "Regular hexagon $P_{1} P_{2} P_{3} P_{4} P_{5} P_{6}$ has side length 2. For $1 \\leq i \\leq 6$, let $C_{i}$ be a unit circle centered at $P_{i}$ and $\\ell_{i}$ be one of the internal common tangents of $C_{i}$ and $C_{i+2}$, where $C_{7}=C_{1}$ and $C_{8}=C_{2}$. Assume that the lines $\\{\\ell_{1}, \\ell_{2}, \\ell_{3}, \\ell_{4}, \\ell_{5}, \\ell_{6}\\}$ bound a regular hexagon. The area of this hexagon can be expressed as $\\sqrt{\\frac{a}{b}}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "1603",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_11",
        "source_index": 2094,
        "difficulty": 5.0
      },
      {
        "id": "7f452480d09f4b6aa956e0677119ba4c920e47580071836efcabf41a1c1e944a",
        "domain": "omni",
        "question": "Let $a \\geq b \\geq c$ be real numbers such that $$\\begin{aligned} a^{2} b c+a b^{2} c+a b c^{2}+8 & =a+b+c \\\\ a^{2} b+a^{2} c+b^{2} c+b^{2} a+c^{2} a+c^{2} b+3 a b c & =-4 \\\\ a^{2} b^{2} c+a b^{2} c^{2}+a^{2} b c^{2} & =2+a b+b c+c a \\end{aligned}$$ If $a+b+c>0$, then compute the integer nearest to $a^{5}$.",
        "answer": "1279",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 571,
        "difficulty": 5.0
      },
      {
        "id": "204cc51da7e97ee31115ac7f699a4763f5de325099d7a3a5b9148ea2e9b8f305",
        "domain": "omni",
        "question": "If $x=3$, $y=2x$, and $z=3y$, what is the value of $z$?",
        "answer": "18",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2700,
        "difficulty": 1.0
      },
      {
        "id": "68cdab7ea5500e4bcf44ff26f35e2dfcb8108b411cd76ff708232d5ca491e282",
        "domain": "omni",
        "question": "Compute the number of ways a non-self-intersecting concave quadrilateral can be drawn in the plane such that two of its vertices are $(0,0)$ and $(1,0)$, and the other two vertices are two distinct lattice points $(a, b),(c, d)$ with $0 \\leq a, c \\leq 59$ and $1 \\leq b, d \\leq 5$.",
        "answer": "366",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1934,
        "difficulty": 5.0
      },
      {
        "id": "2e9a0b2d71f56f1ba5c3a8771ade2e8c42dc151d67c6ef5dc7bc2decb67baa2d",
        "domain": "omni",
        "question": "Find the smallest value that the expression takes $x^4 + y^4 - x^2y - xy^2$, for positive numbers $x$ and $y$ satisfying $x + y \\le 1$.",
        "answer": "-\\frac{1}{8}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "czech-polish-slovak matches",
        "source_index": 4008,
        "difficulty": 5.0
      },
      {
        "id": "c263676224d89695b798df7b750f26d0283a18f2fb4c883833a29049d7eb8fe9",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=2, C A=3, B C=4$. Let $D$ be the point diametrically opposite $A$ on the circumcircle of $A B C$, and let $E$ lie on line $A D$ such that $D$ is the midpoint of $\\overline{A E}$. Line $l$ passes through $E$ perpendicular to $\\overline{A E}$, and $F$ and $G$ are the intersections of the extensions of $\\overline{A B}$ and $\\overline{A C}$ with $l$. Compute $F G$.",
        "answer": "\\frac{1024}{45}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 561,
        "difficulty": 5.25
      },
      {
        "id": "416c8437db33fd196a90eb5248cbb4377dde3cfa62cfaf09abf84a9d2ff8878f",
        "domain": "omni",
        "question": "Determine the number of four-digit integers $n$ such that $n$ and $2n$ are both palindromes.",
        "answer": "20",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Discrete Mathematics -> Other",
        "source": "HMMT_2",
        "source_index": 553,
        "difficulty": 4.0
      },
      {
        "id": "718760a9db041043866800832a7b9ce24a9827b56cc2d7f00e2b4f9da1ddf304",
        "domain": "omni",
        "question": "A digital clock shows the time $4:56$. How many minutes will pass until the clock next shows a time in which all of the digits are consecutive and are in increasing order?",
        "answer": "458",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2757,
        "difficulty": 2.5
      },
      {
        "id": "6e316b8ea7f8b0358a75c0cc9b658ca62a550a29a2b9eba2ba5230f880d280bb",
        "domain": "omni",
        "question": "Suppose that $PQRS TUVW$ is a regular octagon. There are 70 ways in which four of its sides can be chosen at random. If four of its sides are chosen at random and each of these sides is extended infinitely in both directions, what is the probability that they will meet to form a quadrilateral that contains the octagon?",
        "answer": "\\frac{19}{35}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "cayley",
        "source_index": 3147,
        "difficulty": 2.5
      },
      {
        "id": "b689d0931045c8b634eeeb6abb117f315a96d925a3940bb3c20c6ce84404afa2",
        "domain": "omni",
        "question": "A class of 30 students was asked what they did on their winter holiday. 20 students said that they went skating. 9 students said that they went skiing. Exactly 5 students said that they went skating and went skiing. How many students did not go skating and did not go skiing?",
        "answer": "6",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 2688,
        "difficulty": 1.5
      },
      {
        "id": "481a43e5765006b18b1d5c8a780d9488994740c9f3b03af0467e2934edc3a7c7",
        "domain": "omni",
        "question": "A rectangle has width $x$ and length $y$. The rectangle is cut along the horizontal and vertical dotted lines to produce four smaller rectangles. The sum of the perimeters of these four rectangles is 24. What is the value of $x+y$?",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "cayley",
        "source_index": 2715,
        "difficulty": 1.5
      },
      {
        "id": "680599c55d68ed311f58c5e0d5e5e86207a71e1df755042c32052ba2cb901ea7",
        "domain": "omni",
        "question": "Rachel has two indistinguishable tokens, and places them on the first and second square of a $1 \\times 6$ grid of squares. She can move the pieces in two ways: If a token has a free square in front of it, then she can move this token one square to the right. If the square immediately to the right of a token is occupied by the other token, then she can 'leapfrog' the first token; she moves the first token two squares to the right, over the other token, so that it is on the square immediately to the right of the other token. If a token reaches the 6th square, then it cannot move forward any more, and Rachel must move the other one until it reaches the 5th square. How many different sequences of moves for the tokens can Rachel make so that the two tokens end up on the 5th square and the 6th square?",
        "answer": "42",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2113,
        "difficulty": 4.5
      },
      {
        "id": "b9806703111585994a50a52c6c5048222ca570efe41340067f91be76dfe62a37",
        "domain": "omni",
        "question": "If $a_{1}=1, a_{2}=0$, and $a_{n+1}=a_{n}+\\frac{a_{n+2}}{2}$ for all $n \\geq 1$, compute $a_{2004}$.",
        "answer": "-2^{1002}",
        "topic": "Mathematics -> Algebra -> Sequences and Series -> Other",
        "source": "HMMT_2",
        "source_index": 691,
        "difficulty": 4.5
      },
      {
        "id": "2aca87976ab2cb4ce71609c2060252eb36ad563e0d8ae7d0e0778a33ab01cf13",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle whose incircle has center $I$ and is tangent to $\\overline{B C}, \\overline{C A}, \\overline{A B}$, at $D, E, F$. Denote by $X$ the midpoint of major arc $\\widehat{B A C}$ of the circumcircle of $A B C$. Suppose $P$ is a point on line $X I$ such that $\\overline{D P} \\perp \\overline{E F}$. Given that $A B=14, A C=15$, and $B C=13$, compute $D P$.",
        "answer": "\\frac{4 \\sqrt{5}}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 690,
        "difficulty": 5.0
      },
      {
        "id": "2e2b0033d206b5fce1101f374ea226ad52582c477a42f36c4277481992fc1f34",
        "domain": "omni",
        "question": "Two integers are relatively prime if they don't share any common factors, i.e. if their greatest common divisor is 1. Define $\\varphi(n)$ as the number of positive integers that are less than $n$ and relatively prime to $n$. Define $\\varphi_{d}(n)$ as the number of positive integers that are less than $d n$ and relatively prime to $n$. What is the least $n$ such that $\\varphi_{x}(n)=64000$, where $x=\\varphi_{y}(n)$, where $y=\\varphi(n)$?",
        "answer": "41",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 849,
        "difficulty": 5.25
      },
      {
        "id": "9976b1df56a3ad752cb7ffaf905b5b1cb1c8590d929e68dc04b46a2ba61e5969",
        "domain": "omni",
        "question": "What is the remainder when the integer equal to \\( QT^2 \\) is divided by 100, given that \\( QU = 9 \\sqrt{33} \\) and \\( UT = 40 \\)?",
        "answer": "9",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circular and Triangular Relationships -> Other; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "cayley",
        "source_index": 3077,
        "difficulty": 2.0
      },
      {
        "id": "d2ebf3c44fbecb0b7db6cc29aa7faf3a274986807ba84d149fc08bdddbc5ddb1",
        "domain": "omni",
        "question": "Three different numbers from the list $2, 3, 4, 6$ have a sum of 11. What is the product of these numbers?",
        "answer": "36",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2745,
        "difficulty": 2.0
      },
      {
        "id": "ddd1632f9c2681c5dafb1e93dfe0a762d2fb97e776f14d1b03c247b6dc3daca2",
        "domain": "omni",
        "question": "The skeletal structure of coronene, a hydrocarbon with the chemical formula $\\mathrm{C}_{24} \\mathrm{H}_{12}$, is shown below. Each line segment between two atoms is at least a single bond. However, since each carbon (C) requires exactly four bonds connected to it and each hydrogen $(\\mathrm{H})$ requires exactly one bond, some of the line segments are actually double bonds. How many arrangements of single/double bonds are there such that the above requirements are satisfied?",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2638,
        "difficulty": 5.0
      },
      {
        "id": "3f19f161965da0ee79ac2cded0a1dd0858b1ca4221e7e7710d929aa1d4e4132e",
        "domain": "omni",
        "question": "Find the minimum positive integer $k$ such that $f(n+k) \\equiv f(n)(\\bmod 23)$ for all integers $n$.",
        "answer": "2530",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2636,
        "difficulty": 4.0
      },
      {
        "id": "67ecbae5186cc46fe0a55139e26fb69ded54867c951928689db14e79a06674c7",
        "domain": "omni",
        "question": "The product of $N$ consecutive four-digit positive integers is divisible by $2010^{2}$. What is the least possible value of $N$?",
        "answer": "5",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "pascal",
        "source_index": 2818,
        "difficulty": 2.0
      },
      {
        "id": "0acaf662ad09999a52b8d788a68d684bfa4b3546eb5a2f395b5efa261f6a7d69",
        "domain": "omni",
        "question": "Given two distinct points $A, B$ and line $\\ell$ that is not perpendicular to $A B$, what is the maximum possible number of points $P$ on $\\ell$ such that $A B P$ is an isosceles triangle?",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 840,
        "difficulty": 5.0
      },
      {
        "id": "b8252c5573472cac57a85fc77be9fb7f29dc127851e5a207f62d71f736eeba88",
        "domain": "omni",
        "question": "For each positive integer $n$, define $s(n)$ to equal the sum of the digits of $n$. The number of integers $n$ with $100 \\leq n \\leq 999$ and $7 \\leq s(n) \\leq 11$ is $S$. What is the integer formed by the rightmost two digits of $S$?",
        "answer": "24",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "fermat",
        "source_index": 2843,
        "difficulty": 2.5
      },
      {
        "id": "cc9dbfcc2c58b2622faaa76add32c01061aa47d8d9af5c0069a93c873bbfb112",
        "domain": "omni",
        "question": "Let \\(a_{1}, a_{2}, \\ldots\\) be an infinite sequence of integers such that \\(a_{i}\\) divides \\(a_{i+1}\\) for all \\(i \\geq 1\\), and let \\(b_{i}\\) be the remainder when \\(a_{i}\\) is divided by 210. What is the maximal number of distinct terms in the sequence \\(b_{1}, b_{2}, \\ldots\\)?",
        "answer": "127",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 731,
        "difficulty": 5.0
      },
      {
        "id": "9bcc47a214ea2970d8774f64c086bfd4eff8cba0fea1b2c3455c7d2f14cadfef",
        "domain": "omni",
        "question": "Charlie is making a necklace with yellow beads and green beads. She has already used 4 green beads and 0 yellow beads. How many yellow beads will she have to add so that $\frac{4}{5}$ of the total number of beads are yellow?",
        "answer": "16",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 2811,
        "difficulty": 1.5
      },
      {
        "id": "8eec61744167185ab72ad1e32fc75fc151e2e8ce7223ee50a029aae4ec4ea566",
        "domain": "omni",
        "question": "The mathematician John is having trouble remembering his girlfriend Alicia's 7-digit phone number. He remembers that the first four digits consist of one 1, one 2, and two 3s. He also remembers that the fifth digit is either a 4 or 5. While he has no memory of the sixth digit, he remembers that the seventh digit is 9 minus the sixth digit. If this is all the information he has, how many phone numbers does he have to try if he is to make sure he dials the correct number?",
        "answer": "240",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 828,
        "difficulty": 3.5
      },
      {
        "id": "43484dcb74c8ee7e68408c4679549a2da589caec97b5c038e0a873eee6faf2b8",
        "domain": "omni",
        "question": "Compute the number of positive integers that divide at least two of the integers in the set $\\{1^{1}, 2^{2}, 3^{3}, 4^{4}, 5^{5}, 6^{6}, 7^{7}, 8^{8}, 9^{9}, 10^{10}\\}$.",
        "answer": "22",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1438,
        "difficulty": 4.0
      },
      {
        "id": "66a150fe79bb18f64e982b482353b403b36d0820e41962486a1b984df8072798",
        "domain": "omni",
        "question": "If Kai will celebrate his 25th birthday in March 2020, in what year was Kai born?",
        "answer": "1995",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3393,
        "difficulty": 1.5
      },
      {
        "id": "fed6bc5c218c4e89b4b1b16dd90a34fc5d0d5d7413df2ff17165706edf33d897",
        "domain": "omni",
        "question": "Compute $$\\sum_{\\substack{a+b+c=12 \\\\ a \\geq 6, b, c \\geq 0}} \\frac{a!}{b!c!(a-b-c)!}$$ where the sum runs over all triples of nonnegative integers $(a, b, c)$ such that $a+b+c=12$ and $a \\geq 6$.",
        "answer": "2731",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1987,
        "difficulty": 5.0
      },
      {
        "id": "6f1d551b3b6bcfb4a8546f5fd28441b0d66fc0d4f6c10709b2104825d9d6c5ca",
        "domain": "omni",
        "question": "For any finite sequence of positive integers \\pi, let $S(\\pi)$ be the number of strictly increasing subsequences in \\pi with length 2 or more. For example, in the sequence $\\pi=\\{3,1,2,4\\}$, there are five increasing sub-sequences: $\\{3,4\\},\\{1,2\\},\\{1,4\\},\\{2,4\\}$, and $\\{1,2,4\\}$, so $S(\\pi)=5$. In an eight-player game of Fish, Joy is dealt six cards of distinct values, which she puts in a random order \\pi from left to right in her hand. Determine $\\sum_{\\pi} S(\\pi)$ where the sum is taken over all possible orders \\pi of the card values.",
        "answer": "8287",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2197,
        "difficulty": 4.5
      },
      {
        "id": "a51a2f1019c678f53c4f2e9fa2c733306eda2ab52c8899aa26446be27fd6ecaf",
        "domain": "omni",
        "question": "Compute the number of tuples $\\left(a_{0}, a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ of (not necessarily positive) integers such that $a_{i} \\leq i$ for all $0 \\leq i \\leq 5$ and $$a_{0}+a_{1}+\\cdots+a_{5}=6$$",
        "answer": "2002",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1868,
        "difficulty": 5.25
      },
      {
        "id": "175f9a4c499fd5df12caa2a3a70e9bdab7617f48a3e1114610ae78b3ac20724e",
        "domain": "omni",
        "question": "Suppose that point $D$ lies on side $B C$ of triangle $A B C$ such that $A D$ bisects $\\angle B A C$, and let $\\ell$ denote the line through $A$ perpendicular to $A D$. If the distances from $B$ and $C$ to $\\ell$ are 5 and 6 , respectively, compute $A D$.",
        "answer": "\\frac{60}{11}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1812,
        "difficulty": 4.5
      },
      {
        "id": "eb061da5d6ad7596c505cb515683dff2c2b31798966f02d18ad68de2be120ddd",
        "domain": "omni",
        "question": "What is the 3-digit number formed by the $9998^{\\text {th }}$ through $10000^{\\text {th }}$ digits after the decimal point in the decimal expansion of \\frac{1}{998}$ ?",
        "answer": "042",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 1918,
        "difficulty": 4.5
      },
      {
        "id": "d8b4d2ba0826df1519829b040dcd7d8d200b8a7c8b4ddc223d7a827598626564",
        "domain": "omni",
        "question": "Wesyu is a farmer, and she's building a cao (a relative of the cow) pasture. She starts with a triangle $A_{0} A_{1} A_{2}$ where angle $A_{0}$ is $90^{\\circ}$, angle $A_{1}$ is $60^{\\circ}$, and $A_{0} A_{1}$ is 1. She then extends the pasture. First, she extends $A_{2} A_{0}$ to $A_{3}$ such that $A_{3} A_{0}=\\frac{1}{2} A_{2} A_{0}$ and the new pasture is triangle $A_{1} A_{2} A_{3}$. Next, she extends $A_{3} A_{1}$ to $A_{4}$ such that $A_{4} A_{1}=\\frac{1}{6} A_{3} A_{1}$. She continues, each time extending $A_{n} A_{n-2}$ to $A_{n+1}$ such that $A_{n+1} A_{n-2}=\\frac{1}{2^{n}-2} A_{n} A_{n-2}$. What is the smallest $K$ such that her pasture never exceeds an area of $K$?",
        "answer": "\\sqrt{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 575,
        "difficulty": 5.0
      },
      {
        "id": "3c55a18c2d1e3ae73d0cb5c9585f0bc329c0413f445b5f335eca7f6dff786e27",
        "domain": "omni",
        "question": "Suppose $a$ and $b$ are positive integers. Isabella and Vidur both fill up an $a \\times b$ table. Isabella fills it up with numbers $1,2, \\ldots, a b$, putting the numbers $1,2, \\ldots, b$ in the first row, $b+1, b+2, \\ldots, 2 b$ in the second row, and so on. Vidur fills it up like a multiplication table, putting $i j$ in the cell in row $i$ and column $j$. Isabella sums up the numbers in her grid, and Vidur sums up the numbers in his grid; the difference between these two quantities is 1200. Compute $a+b$.",
        "answer": "21",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1626,
        "difficulty": 5.0
      },
      {
        "id": "84915ccd1903357048163982bbb30aa2b08e0637bab90bc875351b3ce5b556c2",
        "domain": "omni",
        "question": "Consider a $4 \\times 4$ grid of squares, each of which are originally colored red. Every minute, Piet can jump on one of the squares, changing the color of it and any adjacent squares (two squares are adjacent if they share a side) to blue. What is the minimum number of minutes it will take Piet to change the entire grid to blue?",
        "answer": "4",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1750,
        "difficulty": 4.5
      },
      {
        "id": "758c0b3226ae43d27acd8e81f5dcb97d31f95e22d45aa33930c32869ca8d1abe",
        "domain": "omni",
        "question": "For how many integers $m$, with $1 \\leq m \\leq 30$, is it possible to find a value of $n$ so that $n!$ ends with exactly $m$ zeros?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Factorization",
        "source": "fermat",
        "source_index": 2978,
        "difficulty": 2.5
      },
      {
        "id": "450f8b4cd7cb155ad60ec36402fb11b6b9f8fe7417e4eb45ac5f07166e8b2914",
        "domain": "omni",
        "question": "An $E$-shape is a geometric figure in the two-dimensional plane consisting of three rays pointing in the same direction, along with a line segment such that the endpoints of the rays all lie on the segment, the segment is perpendicular to all three rays, both endpoints of the segment are endpoints of rays. Suppose two $E$-shapes intersect each other $N$ times in the plane for some positive integer $N$. Compute the maximum possible value of $N$.",
        "answer": "11",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Other",
        "source": "HMMT_2",
        "source_index": 1299,
        "difficulty": 5.25
      },
      {
        "id": "7d7d448e4c4fcf7b137732eabb91c2a99749ce6c94bfda6a09fb163850f8c7da",
        "domain": "omni",
        "question": "Regular octagon CHILDREN has area 1. Determine the area of quadrilateral LINE.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2565,
        "difficulty": 4.5
      },
      {
        "id": "bb36896ce5f2f4d7f8a97151adf8ef8404fb6d62c872344b3aafb02800d5459c",
        "domain": "omni",
        "question": "Compute the value of $1^{25}+2^{24}+3^{23}+\\ldots+24^{2}+25^{1}$. If your answer is $A$ and the correct answer is $C$, then your score on this problem will be $\\left\\lfloor 25 \\mathrm{~min}\\left(\\left(\\frac{A}{C}\\right)^{2},\\left(\\frac{C}{A}\\right)^{2}\\right)\\right\\rfloor$.",
        "answer": "66071772829247409",
        "topic": "Mathematics -> Algebra -> Exponents and Powers -> Other",
        "source": "HMMT_2",
        "source_index": 638,
        "difficulty": 5.0
      },
      {
        "id": "eb94fb63bdb0e67f9c5f02f6ccb03a2329899391938ab5c22cad3372b0f665c6",
        "domain": "omni",
        "question": "You have a length of string and 7 beads in the 7 colors of the rainbow. You place the beads on the string as follows - you randomly pick a bead that you haven't used yet, then randomly add it to either the left end or the right end of the string. What is the probability that, at the end, the colors of the beads are the colors of the rainbow in order? (The string cannot be flipped, so the red bead must appear on the left side and the violet bead on the right side.)",
        "answer": "\\frac{1}{5040}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 2325,
        "difficulty": 4.5
      },
      {
        "id": "1a8a3885de3cf4d4ce3d52962e85f03f83921b1642108ef618e6fa0a3e33075a",
        "domain": "omni",
        "question": "If $x=11, y=8$, and $2x+3z=5y$, what is the value of $z$?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3158,
        "difficulty": 2.5
      },
      {
        "id": "ead1fb3b035ac90956283d3e12a09b9df06f9aca9a17495c2a30a27ea2e53125",
        "domain": "omni",
        "question": "Compute the maximum number of sides of a polygon that is the cross-section of a regular hexagonal prism.",
        "answer": "8",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2453,
        "difficulty": 3.5
      },
      {
        "id": "f3c9d2fa354c474596a2b9fc10da6f54b268d5ef43cdfab7f7f7f698cbfaca0d",
        "domain": "omni",
        "question": "Find the number of pairs $(a, b)$ of positive integers with the property that the greatest common divisor of $a$ and $ b$ is equal to $1\\cdot 2 \\cdot 3\\cdot ... \\cdot50$, and the least common multiple of $a$ and $ b$ is $1^2 \\cdot 2^2 \\cdot 3^2\\cdot ... \\cdot 50^2$.",
        "answer": "32768",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Number Theory -> Least Common Multiples (LCM)",
        "source": "czech-polish-slovak matches",
        "source_index": 3650,
        "difficulty": 6.0
      },
      {
        "id": "532d5ae5a55d1f1b2b9b769a2ec678c54bbb73abbe55fd2c858eb8f57859c0d8",
        "domain": "omni",
        "question": "Morgan uses a spreadsheet to create a table of values. In the first column, she lists the positive integers from 1 to 400. She then puts integers in the second column in the following way: if the integer in the first column of a given row is $n$, the number in the second column of that row is $3 n+1$. Which of the following integers does not appear in the second column: 31, 94, 131, 331, 907?",
        "answer": "131",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 3486,
        "difficulty": 1.0
      },
      {
        "id": "7f5cf1ceb39200f48b30371eb4ba55bb9c107d2bf06180ba8e4506a76ffbc933",
        "domain": "omni",
        "question": "Ben \"One Hunna Dolla\" Franklin is flying a kite KITE such that $I E$ is the perpendicular bisector of $K T$. Let $I E$ meet $K T$ at $R$. The midpoints of $K I, I T, T E, E K$ are $A, N, M, D$, respectively. Given that $[M A K E]=18, I T=10,[R A I N]=4$, find $[D I M E]$.",
        "answer": "16",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2294,
        "difficulty": 4.0
      },
      {
        "id": "f18808edeb9c1ce13db779138255de81135dd56204b2fa7903379400ba372cbc",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle with $A B=13, B C=14$, and $C A=15$. Let $D$ be the foot of the altitude from $A$ to $B C$. The inscribed circles of triangles $A B D$ and $A C D$ are tangent to $A D$ at $P$ and $Q$, respectively, and are tangent to $B C$ at $X$ and $Y$, respectively. Let $P X$ and $Q Y$ meet at $Z$. Determine the area of triangle $X Y Z$.",
        "answer": "\\frac{25}{4}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2082,
        "difficulty": 4.0
      },
      {
        "id": "155ce9bed2adab445b895c9601444beb7e34c4006e91602782d7aa0ae4d6b4c7",
        "domain": "omni",
        "question": "Flat Albert and his buddy Mike are watching the game on Sunday afternoon. Albert is drinking lemonade from a two-dimensional cup which is an isosceles triangle whose height and base measure 9 cm and 6 cm; the opening of the cup corresponds to the base, which points upwards. Every minute after the game begins, the following takes place: if $n$ minutes have elapsed, Albert stirs his drink vigorously and takes a sip of height $\\frac{1}{n^{2}} \\mathrm{~cm}$. Shortly afterwards, while Albert is busy watching the game, Mike adds cranberry juice to the cup until it's once again full in an attempt to create Mike's cranberry lemonade. Albert takes sips precisely every minute, and his first sip is exactly one minute after the game begins. After an infinite amount of time, let $A$ denote the amount of cranberry juice that has been poured (in square centimeters). Find the integer nearest $\\frac{27}{\\pi^{2}} A$.",
        "answer": "26",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1724,
        "difficulty": 5.0
      },
      {
        "id": "9033442a9d3a92e3eb49da76bde397d46197415e12d7e52cd4f00781b1f95c42",
        "domain": "omni",
        "question": "Find the largest positive integer $k{}$ for which there exists a convex polyhedron $\\mathcal{P}$ with 2022 edges, which satisfies the following properties:\n[list]\n[*]The degrees of the vertices of $\\mathcal{P}$ don’t differ by more than one, and\n[*]It is possible to colour the edges of $\\mathcal{P}$ with $k{}$ colours such that for every colour $c{}$, and every pair of vertices $(v_1, v_2)$ of $\\mathcal{P}$, there is a monochromatic path between $v_1$ and $v_2$ in the colour $c{}$.\n[/list]\n[i]Viktor Simjanoski, Macedonia[/i]",
        "answer": "2",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "balkan_mo_shortlist",
        "source_index": 3587,
        "difficulty": 7.0
      },
      {
        "id": "2ae3890499bb3b3fea3951daf58bbe9b308996ab0a405122f96389338284a53e",
        "domain": "omni",
        "question": "Let $f(x)=x^{2}-2 x$. How many distinct real numbers $c$ satisfy $f(f(f(f(c))))=3$ ?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 686,
        "difficulty": 5.0
      },
      {
        "id": "e2d9e658a4514121f24a67b5bc3eb6effaac577a0189e6746578f3d2f2e947a0",
        "domain": "omni",
        "question": "How many of the numbers in Grace's sequence, starting from 43 and each number being 4 less than the previous one, are positive?",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3122,
        "difficulty": 1.0
      },
      {
        "id": "cf3831e907e29121d433f0d8040d677772cb123f7cb4ae0afa6c9ab9cb9122dc",
        "domain": "omni",
        "question": "Sam rolls a fair four-sided die containing the numbers $1,2,3$, and 4. Tyler rolls a fair six-sided die containing the numbers $1,2,3,4,5$, and 6. What is the probability that Sam rolls a larger number than Tyler?",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 2963,
        "difficulty": 2.0
      },
      {
        "id": "2847e1693fe471ccf575ea818de80e19c2a5b78d80134da316eae0140608e5ae",
        "domain": "omni",
        "question": "Let $G_{1} G_{2} G_{3}$ be a triangle with $G_{1} G_{2}=7, G_{2} G_{3}=13$, and $G_{3} G_{1}=15$. Let $G_{4}$ be a point outside triangle $G_{1} G_{2} G_{3}$ so that ray $\\overrightarrow{G_{1} G_{4}}$ cuts through the interior of the triangle, $G_{3} G_{4}=G_{4} G_{2}$, and $\\angle G_{3} G_{1} G_{4}=30^{\\circ}$. Let $G_{3} G_{4}$ and $G_{1} G_{2}$ meet at $G_{5}$. Determine the length of segment $G_{2} G_{5}$.",
        "answer": "\\frac{169}{23}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2200,
        "difficulty": 4.5
      },
      {
        "id": "2a4d03c723f51ccf7922471961082890952c68ad3f0d5a36b1b54ca1fec8f4c8",
        "domain": "omni",
        "question": "Let $S=\\left\\{p_{1} p_{2} \\cdots p_{n} \\mid p_{1}, p_{2}, \\ldots, p_{n}\\right.$ are distinct primes and $\\left.p_{1}, \\ldots, p_{n}<30\\right\\}$. Assume 1 is in $S$. Let $a_{1}$ be an element of $S$. We define, for all positive integers $n$ : $$ \\begin{gathered} a_{n+1}=a_{n} /(n+1) \\quad \\text { if } a_{n} \\text { is divisible by } n+1 \\\\ a_{n+1}=(n+2) a_{n} \\quad \\text { if } a_{n} \\text { is not divisible by } n+1 \\end{gathered} $$ How many distinct possible values of $a_{1}$ are there such that $a_{j}=a_{1}$ for infinitely many $j$ 's?",
        "answer": "512",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 918,
        "difficulty": 5.0
      },
      {
        "id": "74e6685be9fc66ca53569ac617c7a56cfcb48dd82ebe6bd918d22a6da46f1fd1",
        "domain": "omni",
        "question": "Let $\\Delta A_{1} B_{1} C$ be a triangle with $\\angle A_{1} B_{1} C=90^{\\circ}$ and $\\frac{C A_{1}}{C B_{1}}=\\sqrt{5}+2$. For any $i \\geq 2$, define $A_{i}$ to be the point on the line $A_{1} C$ such that $A_{i} B_{i-1} \\perp A_{1} C$ and define $B_{i}$ to be the point on the line $B_{1} C$ such that $A_{i} B_{i} \\perp B_{1} C$. Let $\\Gamma_{1}$ be the incircle of $\\Delta A_{1} B_{1} C$ and for $i \\geq 2, \\Gamma_{i}$ be the circle tangent to $\\Gamma_{i-1}, A_{1} C, B_{1} C$ which is smaller than $\\Gamma_{i-1}$. How many integers $k$ are there such that the line $A_{1} B_{2016}$ intersects $\\Gamma_{k}$ ?",
        "answer": "4030",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1088,
        "difficulty": 6.0
      },
      {
        "id": "8944e3602f703a0fc25eeb900032b2b27f79483143ea54122baba59a4b1e2b75",
        "domain": "omni",
        "question": "Let $a, b$ and $c$ be positive real numbers such that $$\\begin{aligned} a^{2}+a b+b^{2} & =9 \\\\ b^{2}+b c+c^{2} & =52 \\\\ c^{2}+c a+a^{2} & =49 \\end{aligned}$$ Compute the value of $\\frac{49 b^{2}-33 b c+9 c^{2}}{a^{2}}$.",
        "answer": "52",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 1345,
        "difficulty": 5.0
      },
      {
        "id": "e6cd7b51a7a3d31541ce65dae6a0faf8963039a436e4eb0716e1811b74fb205c",
        "domain": "omni",
        "question": "Let $\\{ z_n \\}_{n \\ge 1}$ be a sequence of complex numbers, whose odd terms are real, even terms are purely imaginary, and for every positive integer $k$, $|z_k z_{k+1}|=2^k$. Denote $f_n=|z_1+z_2+\\cdots+z_n|,$ for $n=1,2,\\cdots$\n(1) Find the minimum of $f_{2020}$.\n(2) Find the minimum of $f_{2020} \\cdot f_{2021}$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Number Theory -> Other; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_national_olympiad",
        "source_index": 49,
        "difficulty": 8.0
      },
      {
        "id": "8af1d08336e9c6cd9c9b49b76ca45083a2389b73cc9b0ccaa541a53d5d168149",
        "domain": "omni",
        "question": "Mario is once again on a quest to save Princess Peach. Mario enters Peach's castle and finds himself in a room with 4 doors. This room is the first in a sequence of 6 indistinguishable rooms. In each room, 1 door leads to the next room in the sequence (or, for the last room, Bowser's level), while the other 3 doors lead to the first room. Now what is the expected number of doors through which Mario will pass before he reaches Bowser's level?",
        "answer": "5460",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1727,
        "difficulty": 5.0
      },
      {
        "id": "97987d67ba093145938ea0fb978f65a5f4267babfe6c65df557d4548a73dde7d",
        "domain": "omni",
        "question": "Find the value of $\\frac{\\sin^{2}B+\\sin^{2}C-\\sin^{2}A}{\\sin B \\sin C}$ given that $\\frac{\\sin B}{\\sin C}=\\frac{AC}{AB}$, $\\frac{\\sin C}{\\sin B}=\\frac{AB}{AC}$, and $\\frac{\\sin A}{\\sin B \\sin C}=\\frac{BC}{AC \\cdot AB}$.",
        "answer": "\\frac{83}{80}",
        "topic": "Mathematics -> Precalculus -> Trigonometric Functions; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2410,
        "difficulty": 5.25
      },
      {
        "id": "096d91ab0f3a83d2630d50d1a869e6ca69bffa41050366d1d9219a464be6fece",
        "domain": "omni",
        "question": "Let $f(x)=-x^{2}+10 x-20$. Find the sum of all $2^{2010}$ solutions to $\\underbrace{f(f(\\ldots(x) \\ldots))}_{2010 f \\mathrm{~s}}=2$.",
        "answer": "5 \\cdot 2^{2010}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Precalculus -> Functions",
        "source": "HMMT_11",
        "source_index": 2635,
        "difficulty": 5.25
      },
      {
        "id": "b8ac78cf23b764253f423542399f45a2ae803442149a5947c61a200dca823afd",
        "domain": "omni",
        "question": "Consider a $2 \\times 2$ grid of squares. Each of the squares will be colored with one of 10 colors, and two colorings are considered equivalent if one can be rotated to form the other. How many distinct colorings are there?",
        "answer": "2530",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1751,
        "difficulty": 5.0
      },
      {
        "id": "12fffab70ed5e1c0ab82ca52c0916324a0c9e3657382a650dbe37eea77dd055b",
        "domain": "omni",
        "question": "Consider triangle $A B C$ where $B C=7, C A=8$, and $A B=9$. $D$ and $E$ are the midpoints of $B C$ and $C A$, respectively, and $A D$ and $B E$ meet at $G$. The reflection of $G$ across $D$ is $G^{\\prime}$, and $G^{\\prime} E$ meets $C G$ at $P$. Find the length $P G$.",
        "answer": "\\frac{\\sqrt{145}}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2209,
        "difficulty": 5.0
      },
      {
        "id": "97efa90d11edf95853ec8f35d284c2afa0fa370095b17e6f05fe30c22912a47e",
        "domain": "omni",
        "question": "In each square of a garden shaped like a $2022 \\times 2022$ board, there is initially a tree of height $0$. A gardener and a lumberjack alternate turns playing the following game, with the gardener taking the first turn:\n[list]\n[*] The gardener chooses a square in the garden. Each tree on that square and all the surrounding squares (of which there are at most eight) then becomes one unit taller.\n[*] The lumberjack then chooses four different squares on the board. Each tree of positive height on those squares then becomes one unit shorter.\n[/list]\nWe say that a tree is [i]majestic[/i] if its height is at least $10^6$. Determine the largest $K$ such that the gardener can ensure there are eventually $K$ majestic trees on the board, no matter how the lumberjack plays.",
        "answer": "2271380",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 3955,
        "difficulty": 9.5
      },
      {
        "id": "f5b28abb2a9a395c094a565cb36af7ce75b1af3774a7a8c2e0b923683e7dd127",
        "domain": "omni",
        "question": "Real numbers $x, y, z$ satisfy $$x+x y+x y z=1, \\quad y+y z+x y z=2, \\quad z+x z+x y z=4$$ The largest possible value of $x y z$ is $\\frac{a+b \\sqrt{c}}{d}$, where $a, b, c, d$ are integers, $d$ is positive, $c$ is square-free, and $\\operatorname{gcd}(a, b, d)=1$. Find $1000 a+100 b+10 c+d$.",
        "answer": "5272",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2577,
        "difficulty": 6.0
      },
      {
        "id": "21bedd358f11573b784d5ada2992b124273bee5b81d7ef831b4a48b20a2fe8ea",
        "domain": "omni",
        "question": "Complex numbers $a, b, c$ form an equilateral triangle with side length 18 in the complex plane. If $|a+b+c|=36$, find $|b c+c a+a b|$.",
        "answer": "432",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 834,
        "difficulty": 4.5
      },
      {
        "id": "a3c47835e3963fa2abf6963ad8d80c68ddaa1e4e7c53f8ec72d882e51d93200f",
        "domain": "omni",
        "question": "Jude repeatedly flips a coin. If he has already flipped $n$ heads, the coin lands heads with probability $\\frac{1}{n+2}$ and tails with probability $\\frac{n+1}{n+2}$. If Jude continues flipping forever, let $p$ be the probability that he flips 3 heads in a row at some point. Compute $\\lfloor 180 p\\rfloor$.",
        "answer": "47",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1211,
        "difficulty": 5.0
      },
      {
        "id": "a9ef3e9e444da8fe1d2eab2bd62ca5c03f4ddc405777cef899f15dd69af82f2b",
        "domain": "omni",
        "question": "A mathematician $M^{\\prime}$ is called a descendent of mathematician $M$ if there is a sequence of mathematicians $M=M_{1}, M_{2}, \\ldots, M_{k}=M^{\\prime}$ such that $M_{i}$ was $M_{i+1}$ 's doctoral advisor for all $i$. Estimate the number of descendents that the mathematician who has had the largest number of descendents has had, according to the Mathematical Genealogy Project. Note that the Mathematical Genealogy Project has records dating back to the 1300s. If the correct answer is $X$ and you write down $A$, your team will receive $\\max \\left(25-\\left\\lfloor\\frac{|X-A|}{100}\\right\\rfloor, 0\\right)$ points, where $\\lfloor x\\rfloor$ is the largest integer less than or equal to $x$.",
        "answer": "82310",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 2601,
        "difficulty": 3.5
      },
      {
        "id": "737cf913309d3f956ddcce3393e00eaf77291f572d79eceddbd6d38b25aebf97",
        "domain": "omni",
        "question": "Three real numbers $x, y, z$ are chosen randomly, and independently of each other, between 0 and 1, inclusive. What is the probability that each of $x-y$ and $x-z$ is greater than $-\\frac{1}{2}$ and less than $\\frac{1}{2}$?",
        "answer": "\\frac{7}{12}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "fermat",
        "source_index": 3056,
        "difficulty": 2.5
      },
      {
        "id": "678e224f5ce791d4786e4e6054c1be745b81d1bfdecc56c93da098da27440014",
        "domain": "omni",
        "question": "Seven lattice points form a convex heptagon with all sides having distinct lengths. Find the minimum possible value of the sum of the squares of the sides of the heptagon.",
        "answer": "42",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2075,
        "difficulty": 4.0
      },
      {
        "id": "82e6caa4181eadc5c8b65bb8bf501f6a9f1e2db0e280433d4432745c2ee98d81",
        "domain": "omni",
        "question": "Zebadiah has 3 red shirts, 3 blue shirts, and 3 green shirts in a drawer. Without looking, he randomly pulls shirts from his drawer one at a time. What is the minimum number of shirts that Zebadiah has to pull out to guarantee that he has a set of shirts that includes either 3 of the same colour or 3 of different colours?",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "fermat",
        "source_index": 2788,
        "difficulty": 2.5
      },
      {
        "id": "8d2c6f63a9a4561912f79564f8e6ee4a96cedb4c28cf512911d5202829479f26",
        "domain": "omni",
        "question": "Calculate the value of $\\frac{2 \\times 3 + 4}{2 + 3}$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 2707,
        "difficulty": 1.5
      },
      {
        "id": "0c43a0c18217c40e6d1f07d76dfa38e82c5b842adb7e2b511d8579060d963bcf",
        "domain": "omni",
        "question": "A candy company makes 5 colors of jellybeans, which come in equal proportions. If I grab a random sample of 5 jellybeans, what is the probability that I get exactly 2 distinct colors?",
        "answer": "\\frac{12}{125}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1509,
        "difficulty": 3.5
      },
      {
        "id": "2b6f340b7d37d1b54b75001c95392fdd4b736154b1bf6497a87c11d50af0a64c",
        "domain": "omni",
        "question": "Compute the nearest integer to $$100 \\sum_{n=1}^{\\infty} 3^{n} \\sin ^{3}\\left(\\frac{\\pi}{3^{n}}\\right)$$",
        "answer": "236",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "HMMT_2",
        "source_index": 1416,
        "difficulty": 4.75
      },
      {
        "id": "1beea403f4ec1fe3a8045284017372b98145773bd2897f0296212ae8fd93a661",
        "domain": "omni",
        "question": "Find the number of strictly increasing sequences of nonnegative integers with the following properties: - The first term is 0 and the last term is 12. In particular, the sequence has at least two terms. - Among any two consecutive terms, exactly one of them is even.",
        "answer": "144",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2292,
        "difficulty": 4.5
      },
      {
        "id": "91fdb4af9c3d3fe942a5f0af08e81dbce14ddf16af61edc9515278cd693ae0c2",
        "domain": "omni",
        "question": "Among a group of 120 people, some pairs are friends. A [i]weak quartet[/i] is a set of four people containing exactly one pair of friends. What is the maximum possible number of weak quartets ?",
        "answer": "4769280",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imo_shortlist",
        "source_index": 3945,
        "difficulty": 7.0
      },
      {
        "id": "7cb9b8c35b1ac0f346ca1c9cd1e2d8a03955069c2baab55901687fefef767a35",
        "domain": "omni",
        "question": "Suppose $P(x)$ is a polynomial with real coefficients such that $P(t)=P(1) t^{2}+P(P(1)) t+P(P(P(1)))$ for all real numbers $t$. Compute the largest possible value of $P(P(P(P(1))))$.",
        "answer": "\\frac{1}{9}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1259,
        "difficulty": 5.0
      },
      {
        "id": "b6a9a8d9590ddb0e355113df94cdc3165401ece55fc90123652ad8b2af6b5886",
        "domain": "omni",
        "question": "Compute $\\sum_{k=1}^{2009} k\\left(\\left\\lfloor\\frac{2009}{k}\\right\\rfloor-\\left\\lfloor\\frac{2008}{k}\\right\\rfloor\\right)$.",
        "answer": "2394",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1423,
        "difficulty": 4.75
      },
      {
        "id": "1ff71d26c02592d13b2638027d0ec2c87b06b3c4d587667dfeaa6a347483373c",
        "domain": "omni",
        "question": "Find the number of quadruples $(a, b, c, d)$ of integers with absolute value at most 5 such that $\\left(a^{2}+b^{2}+c^{2}+d^{2}\\right)^{2}=(a+b+c+d)(a-b+c-d)\\left((a-c)^{2}+(b-d)^{2}\\right)$",
        "answer": "49",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2032,
        "difficulty": 5.0
      },
      {
        "id": "d357801b7f0be8b8ccb6df54a1bd4074164f34b3e4741d37e80e1d9c5a11de81",
        "domain": "omni",
        "question": "Three faces $\\mathcal{X}, \\mathcal{Y}, \\mathcal{Z}$ of a unit cube share a common vertex. Suppose the projections of $\\mathcal{X}, \\mathcal{Y}, \\mathcal{Z}$ onto a fixed plane $\\mathcal{P}$ have areas $x, y, z$, respectively. If $x: y: z=6: 10: 15$, then $x+y+z$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "3119",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2222,
        "difficulty": 4.0
      },
      {
        "id": "aae4b5d0cf84582dc9b326e8eb08046cebe725cce8d908d815f7a752a8e8a03f",
        "domain": "omni",
        "question": "In the base 10 arithmetic problem $H M M T+G U T S=R O U N D$, each distinct letter represents a different digit, and leading zeroes are not allowed. What is the maximum possible value of $R O U N D$?",
        "answer": "16352",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 968,
        "difficulty": 5.0
      },
      {
        "id": "5a3607df364a28ea40c79c8331582aec382da1b5267e878f5e261e253fa119aa",
        "domain": "omni",
        "question": "Suppose $a$ and $b$ be positive integers not exceeding 100 such that $$a b=\\left(\\frac{\\operatorname{lcm}(a, b)}{\\operatorname{gcd}(a, b)}\\right)^{2}$$ Compute the largest possible value of $a+b$.",
        "answer": "78",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2040,
        "difficulty": 4.0
      },
      {
        "id": "bc9249364c1f4a6ff27da9ce396b1518a363fc19f3c58c0f6cb2a9be1a705be7",
        "domain": "omni",
        "question": "The operation $\\nabla$ is defined by $a \\nabla b=4 a+b$. What is the value of $(5 \\nabla 2) \\nabla 2$?",
        "answer": "90",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3050,
        "difficulty": 1.0
      },
      {
        "id": "104bd4a904e8236512409b80775fb5c9c930ea0c3fe09d425fa0c5913bca0554",
        "domain": "omni",
        "question": "Let $N$ be the number of distinct roots of \\prod_{k=1}^{2012}\\left(x^{k}-1\\right)$. Give lower and upper bounds $L$ and $U$ on $N$. If $0<L \\leq N \\leq U$, then your score will be \\left[\\frac{23}{(U / L)^{1.7}}\\right\\rfloor$. Otherwise, your score will be 0 .",
        "answer": "1231288",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 3324,
        "difficulty": 5.0
      },
      {
        "id": "7749d9cef57ada9d90e38ce31f64e40b38c3e85816b3470b65758a27d6532909",
        "domain": "omni",
        "question": "A Sudoku matrix is defined as a $9 \\times 9$ array with entries from \\{1,2, \\ldots, 9\\} and with the constraint that each row, each column, and each of the nine $3 \\times 3$ boxes that tile the array contains each digit from 1 to 9 exactly once. A Sudoku matrix is chosen at random (so that every Sudoku matrix has equal probability of being chosen). We know two of squares in this matrix, as shown. What is the probability that the square marked by ? contains the digit 3 ?",
        "answer": "\\frac{2}{21}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1627,
        "difficulty": 4.0
      },
      {
        "id": "20f38bed9caeb58f4fc0f6a62480ccf204f0d6853d195ab6a4b7aa478111a320",
        "domain": "omni",
        "question": "The integers \\(1,2,3,4,5,6,7,8,9,10\\) are written on a blackboard. Each day, a teacher chooses one of the integers uniformly at random and decreases it by 1. Let \\(X\\) be the expected value of the number of days which elapse before there are no longer positive integers on the board. Estimate \\(X\\). An estimate of \\(E\\) earns \\(\\left\\lfloor 20 \\cdot 2^{-|X-E| / 8}\\right\\rfloor\\) points.",
        "answer": "120.75280458176904",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1611,
        "difficulty": 4.5
      },
      {
        "id": "e72871e6d8522488354872b85813a79d06550482afa9ed9f03fbab26477a8fef",
        "domain": "omni",
        "question": "Let $A_{1} A_{2} \\ldots A_{100}$ be the vertices of a regular 100-gon. Let $\\pi$ be a randomly chosen permutation of the numbers from 1 through 100. The segments $A_{\\pi(1)} A_{\\pi(2)}, A_{\\pi(2)} A_{\\pi(3)}, \\ldots, A_{\\pi(99)} A_{\\pi(100)}, A_{\\pi(100)} A_{\\pi(1)}$ are drawn. Find the expected number of pairs of line segments that intersect at a point in the interior of the 100-gon.",
        "answer": "\\frac{4850}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2016,
        "difficulty": 4.5
      },
      {
        "id": "1d8901b9930e66ea4aa714bf075d2aea11a925504a8df73d72570cef75ffff35",
        "domain": "omni",
        "question": "Find the number of sequences $a_{1}, a_{2}, \\ldots, a_{10}$ of positive integers with the property that $a_{n+2}=a_{n+1}+a_{n}$ for $n=1,2, \\ldots, 8$, and $a_{10}=2002$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 891,
        "difficulty": 4.5
      },
      {
        "id": "91c8815f2918dcea57770cef97b775e371575b080df53182dcd6178b82439b76",
        "domain": "omni",
        "question": "Given that $a, b, c$ are positive real numbers and $\\log _{a} b+\\log _{b} c+\\log _{c} a=0$, find the value of $\\left(\\log _{a} b\\right)^{3}+\\left(\\log _{b} c\\right)^{3}+\\left(\\log _{c} a\\right)^{3}$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_2",
        "source_index": 919,
        "difficulty": 6.0
      },
      {
        "id": "ec495e9c45afa4b111594394b56fb0402396f56c27ad8c1676fa32066f5d5203",
        "domain": "omni",
        "question": "Two numbers $a$ and $b$ with $0 \\leq a \\leq 1$ and $0 \\leq b \\leq 1$ are chosen at random. The number $c$ is defined by $c=2a+2b$. The numbers $a, b$ and $c$ are each rounded to the nearest integer to give $A, B$ and $C$, respectively. What is the probability that $2A+2B=C$?",
        "answer": "\\frac{7}{16}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 3138,
        "difficulty": 2.0
      },
      {
        "id": "1c6321f7cb0c0f4c9f175cb30dd81cbbb03af6fe213ca4deb48ae8abce333907",
        "domain": "omni",
        "question": "A box contains 5 black ties, 7 gold ties, and 8 pink ties. What is the probability that Stephen randomly chooses a pink tie?",
        "answer": "\\frac{2}{5}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "pascal",
        "source_index": 2997,
        "difficulty": 1.0
      },
      {
        "id": "b0b305d6a7a2dd1220e559a178b3b1c5f73f70b962cb4047e88e9c56a3cc0bef",
        "domain": "omni",
        "question": "Lauren plays basketball with her friends. She makes 10 baskets. Each of these baskets is worth either 2 or 3 points. Lauren scores a total of 26 points. How many 3 point baskets did she make?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3198,
        "difficulty": 1.5
      },
      {
        "id": "59d3d54f28c66831f50d69d127c7085d19c18442a460965461efcd1edbd81123",
        "domain": "omni",
        "question": "Let $D$ be the set of divisors of 100. Let $Z$ be the set of integers between 1 and 100, inclusive. Mark chooses an element $d$ of $D$ and an element $z$ of $Z$ uniformly at random. What is the probability that $d$ divides $z$?",
        "answer": "\\frac{217}{900}",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 897,
        "difficulty": 3.5
      },
      {
        "id": "e439f82b2061632e7ef37b34d81608e660397c68c55e7c44b13ea0d4566b6c18",
        "domain": "omni",
        "question": "Let $S$ be the set of ordered pairs $(a, b)$ of positive integers such that \\operatorname{gcd}(a, b)=1$. Compute $$\\sum_{(a, b) \\in S}\\left\\lfloor\\frac{300}{2 a+3 b}\\right\\rfloor$$",
        "answer": "7400",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 1238,
        "difficulty": 5.0
      },
      {
        "id": "1e36d8535b5921ce795cbd58f3a4618e9a1bab455b29151de8db4e50cdd7d40c",
        "domain": "omni",
        "question": "Given any two positive real numbers $x$ and $y$, then $x \\diamond y$ is a positive real number defined in terms of $x$ and $y$ by some fixed rule. Suppose the operation $x \\diamond y$ satisfies the equations \\((x \\cdot y) \\diamond y=x(y \\diamond y)\\) and \\((x \\diamond 1) \\diamond x=x \\diamond 1\\) for all $x, y>0$. Given that $1 \\diamond 1=1$, find $19 \\diamond 98$.",
        "answer": "19",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1443,
        "difficulty": 4.0
      },
      {
        "id": "3d011302d86ea8bc9f6df16e19821bc3435709d49c958399c7f9292d58896278",
        "domain": "omni",
        "question": "The expression $(5 \\times 5)+(5 \\times 5)+(5 \\times 5)+(5 \\times 5)+(5 \\times 5)$ is equal to what?",
        "answer": "125",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 3438,
        "difficulty": 1.5
      },
      {
        "id": "ab69a238793bb1b993ba10facbe96df04ff116cc6808f0523915cd65a5f23d41",
        "domain": "omni",
        "question": "For how many integers $a(1 \\leq a \\leq 200)$ is the number $a^{a}$ a square?",
        "answer": "107",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 752,
        "difficulty": 5.0
      },
      {
        "id": "5d1d8e6ab3f9d929104adb61186c3dd157790db6e970d37a3925d0593ff9e7f2",
        "domain": "omni",
        "question": "Let $a_{1}=3$, and for $n \\geq 1$, let $a_{n+1}=(n+1) a_{n}-n$. Find the smallest $m \\geq 2005$ such that $a_{m+1}-1 \\mid a_{m}^{2}-1$.",
        "answer": "2010",
        "topic": "Mathematics -> Algebra -> Sequences -> Other; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1030,
        "difficulty": 5.5
      },
      {
        "id": "f844c92ee260b2a2b5b87fc8919beb55a4d5873ef338908cf741a8b98a8f30b7",
        "domain": "omni",
        "question": "If points $P, Q, R$, and $S$ are arranged in order on a line segment with $P Q=1, Q R=2 P Q$, and $R S=3 Q R$, what is the length of $P S$?",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3134,
        "difficulty": 1.5
      },
      {
        "id": "686cd63a23edf6588a27fa5606779a99d1141bfb8b1de10be3c9eaca69dcaeb4",
        "domain": "omni",
        "question": "Find the number of pairs of integers $(x, y)$ such that $x^{2}+2y^{2}<25$.",
        "answer": "55",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 436,
        "difficulty": 4.5
      },
      {
        "id": "a7a2a3358609f2f837747409d914db8a8806eea79b094cbbbbd7e42ca50acf1b",
        "domain": "omni",
        "question": "Let $\\varphi(n)$ denote the number of positive integers less than or equal to $n$ which are relatively prime to $n$. Let $S$ be the set of positive integers $n$ such that $\\frac{2 n}{\\varphi(n)}$ is an integer. Compute the sum $\\sum_{n \\in S} \\frac{1}{n}$.",
        "answer": "\\frac{10}{3}",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 513,
        "difficulty": 5.0
      },
      {
        "id": "2fe15404fce70389a966cefce70763c56d985fbb2224941a358100666e0ece71",
        "domain": "omni",
        "question": "Side $\\overline{A B}$ of $\\triangle A B C$ is the diameter of a semicircle, as shown below. If $A B=3+\\sqrt{3}, B C=3 \\sqrt{2}$, and $A C=2 \\sqrt{3}$, then the area of the shaded region can be written as $\\frac{a+(b+c \\sqrt{d}) \\pi}{e}$, where $a, b, c, d, e$ are integers, $e$ is positive, $d$ is square-free, and $\\operatorname{gcd}(a, b, c, e)=1$. Find $10000 a+1000 b+100 c+10 d+e$.",
        "answer": "147938",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Precalculus -> Trigonometric Functions",
        "source": "HMMT_11",
        "source_index": 2639,
        "difficulty": 5.0
      },
      {
        "id": "6151000111586e103628338f47fb3826dd84d7c431835a4694a6b7d6211a004f",
        "domain": "omni",
        "question": "Determine the least possible value of $f(1998),$ where $f:\\Bbb{N}\\to \\Bbb{N}$ is a function such that for all $m,n\\in {\\Bbb N}$, \n\n\\[f\\left( n^{2}f(m)\\right) =m\\left( f(n)\\right) ^{2}. \\]",
        "answer": "120",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "imo",
        "source_index": 3973,
        "difficulty": 9.0
      },
      {
        "id": "22a2b988af0005611b873bbc8d3310f7b312ad0263a2855674fd3c85a07528b9",
        "domain": "omni",
        "question": "If $4^{n}=64^{2}$, what is the value of $n$?",
        "answer": "6",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Exponents -> Other",
        "source": "pascal",
        "source_index": 3006,
        "difficulty": 1.0
      },
      {
        "id": "003869b8f66a62e4249aa5bc013305eeaf67036ca6798c26f6129542fd703ce4",
        "domain": "omni",
        "question": "Let $n$ be an integer and $$m=(n-1001)(n-2001)(n-2002)(n-3001)(n-3002)(n-3003)$$ Given that $m$ is positive, find the minimum number of digits of $m$.",
        "answer": "11",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2655,
        "difficulty": 6.0
      },
      {
        "id": "298b8880436492016194e76993ab087211bb26a5de7e6b3828312f90ac413e57",
        "domain": "omni",
        "question": "Pascal High School organized three different trips. Fifty percent of the students went on the first trip, $80 \\%$ went on the second trip, and $90 \\%$ went on the third trip. A total of 160 students went on all three trips, and all of the other students went on exactly two trips. How many students are at Pascal High School?",
        "answer": "800",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2982,
        "difficulty": 1.5
      },
      {
        "id": "4089d25f7cb4bd311dcabadf7404917c447a97a6fd6e077a06fec12e024a59c7",
        "domain": "omni",
        "question": "A man named Juan has three rectangular solids, each having volume 128. Two of the faces of one solid have areas 4 and 32. Two faces of another solid have areas 64 and 16. Finally, two faces of the last solid have areas 8 and 32. What is the minimum possible exposed surface area of the tallest tower Juan can construct by stacking his solids one on top of the other, face to face? (Assume that the base of the tower is not exposed).",
        "answer": "688",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1213,
        "difficulty": 4.5
      },
      {
        "id": "f3ca7c7dffac8e95691b7e47fe224cfe529a1c38dd14707186a4f403a41ea33b",
        "domain": "omni",
        "question": "Determine the remainder when $$2^{\\frac{1 \\cdot 2}{2}}+2^{\\frac{2 \\cdot 3}{2}}+\\cdots+2^{\\frac{2011 \\cdot 2012}{2}}$$ is divided by 7.",
        "answer": "1",
        "topic": "Mathematics -> Number Theory -> Modular Arithmetic -> Other",
        "source": "HMMT_11",
        "source_index": 2650,
        "difficulty": 4.0
      },
      {
        "id": "cadf7d9ebebe7716e5996f2ca47f2c0f4a68a054853507ffae1d1e989728653e",
        "domain": "omni",
        "question": "Meghal is playing a game with 2016 rounds $1,2, \\cdots, 2016$. In round $n$, two rectangular double-sided mirrors are arranged such that they share a common edge and the angle between the faces is $\\frac{2 \\pi}{n+2}$. Meghal shoots a laser at these mirrors and her score for the round is the number of points on the two mirrors at which the laser beam touches a mirror. What is the maximum possible score Meghal could have after she finishes the game?",
        "answer": "1019088",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_11",
        "source_index": 2101,
        "difficulty": 4.5
      },
      {
        "id": "b95ef98151b6365a824f013728b80708c872bbdafa6aefcf17b6084a0d2ce7b4",
        "domain": "omni",
        "question": "Define the sequence $a_{1}, a_{2} \\ldots$ as follows: $a_{1}=1$ and for every $n \\geq 2$, $a_{n}= \\begin{cases}n-2 & \\text { if } a_{n-1}=0 \\\\ a_{n-1}-1 & \\text { if } a_{n-1} \\neq 0\\end{cases}$. A non-negative integer $d$ is said to be jet-lagged if there are non-negative integers $r, s$ and a positive integer $n$ such that $d=r+s$ and that $a_{n+r}=a_{n}+s$. How many integers in $\\{1,2, \\ldots, 2016\\}$ are jet-lagged?",
        "answer": "51",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1158,
        "difficulty": 4.0
      },
      {
        "id": "d9d92ba66d9d5f7900127ca7158d98e9574f97f03bab3b7d54391e4387fc48d4",
        "domain": "omni",
        "question": "Triangle $A B C$ has incircle $\\omega$ which touches $A B$ at $C_{1}, B C$ at $A_{1}$, and $C A$ at $B_{1}$. Let $A_{2}$ be the reflection of $A_{1}$ over the midpoint of $B C$, and define $B_{2}$ and $C_{2}$ similarly. Let $A_{3}$ be the intersection of $A A_{2}$ with $\\omega$ that is closer to $A$, and define $B_{3}$ and $C_{3}$ similarly. If $A B=9, B C=10$, and $C A=13$, find \\left[A_{3} B_{3} C_{3}\\right] /[A B C].",
        "answer": "14/65",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 1086,
        "difficulty": 5.25
      },
      {
        "id": "670844b0edc703ef03edd6cfa701836bc3dce32caf7ecaf06a1d9002a9faadee",
        "domain": "omni",
        "question": "When $x=-2$, what is the value of $(x+1)^{3}$?",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2855,
        "difficulty": 1.0
      },
      {
        "id": "c9c3329f6bb0504106503b76c1b5b34fb155932dbc46aea567b9e89cb2823d01",
        "domain": "omni",
        "question": "A classroom consists of a $5 \\times 5$ array of desks, to be filled by anywhere from 0 to 25 students, inclusive. No student will sit at a desk unless either all other desks in its row or all others in its column are filled (or both). Considering only the set of desks that are occupied (and not which student sits at each desk), how many possible arrangements are there?",
        "answer": "962",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1292,
        "difficulty": 3.0
      },
      {
        "id": "d485d1449f81c5dccdc79748a60126773507f4e925690e6cb68324c91b9713a9",
        "domain": "omni",
        "question": "Max repeatedly throws a fair coin in a hurricane. For each throw, there is a $4 \\%$ chance that the coin gets blown away. He records the number of heads $H$ and the number of tails $T$ before the coin is lost. (If the coin is blown away on a toss, no result is recorded for that toss.) What is the expected value of $|H-T|$?",
        "answer": "\\frac{24}{7}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1165,
        "difficulty": 4.0
      },
      {
        "id": "42b31be5bd2750ee56841f6a60199811f6c2b541b3bdcea86d315c2b5e10decd",
        "domain": "omni",
        "question": "The vertices of a regular nonagon are colored such that 1) adjacent vertices are different colors and 2) if 3 vertices form an equilateral triangle, they are all different colors. Let m be the minimum number of colors needed for a valid coloring, and n be the total number of colorings using m colors. Determine mn. (Assume each vertex is distinguishable.)",
        "answer": "54",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2159,
        "difficulty": 4.0
      },
      {
        "id": "7fc73c92a3d4603cc1f8bf8f674a6b0f9b05bb1440ec56666d7c2a9f5a16e687",
        "domain": "omni",
        "question": "Let $m \\circ n=(m+n) /(m n+4)$. Compute $((\\cdots((2005 \\circ 2004) \\circ 2003) \\circ \\cdots \\circ 1) \\circ 0)$.",
        "answer": "1/12",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 975,
        "difficulty": 3.5
      },
      {
        "id": "f12948b03ca094700a2650ec076013a004ad0934e41cde18ae249d5624d8cb4d",
        "domain": "omni",
        "question": "A right triangle has area 5 and a hypotenuse of length 5. Find its perimeter.",
        "answer": "5+3 \\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2245,
        "difficulty": 4.0
      },
      {
        "id": "3c9ad03f835f5b071e8288a09a0f114243de6762cb6eea5e411af6b2bf3e985e",
        "domain": "omni",
        "question": "Determine the number of subsets $S$ of $\\{1,2,3, \\ldots, 10\\}$ with the following property: there exist integers $a<b<c$ with $a \\in S, b \\notin S, c \\in S$.",
        "answer": "968",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 915,
        "difficulty": 4.0
      },
      {
        "id": "a85117e4769de4d3c7fefe56a79f897c151949ba730fc2416d9c7959c1a58166",
        "domain": "omni",
        "question": "Solve for \\(x\\): \\(x\\lfloor x\\lfloor x\\lfloor x\\lfloor x\\rfloor\\rfloor\\rfloor\\rfloor=122\\).",
        "answer": "\\frac{122}{41}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 3360,
        "difficulty": 5.0
      },
      {
        "id": "a8130a1d5fc98c1717eea1dcac9d088df0a0ac001d167f01b0e7b7c8f4e0e408",
        "domain": "omni",
        "question": "Let $S$ be a subset of $\\{1,2,3, \\ldots, 12\\}$ such that it is impossible to partition $S$ into $k$ disjoint subsets, each of whose elements sum to the same value, for any integer $k \\geq 2$. Find the maximum possible sum of the elements of $S$.",
        "answer": "77",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2299,
        "difficulty": 5.0
      },
      {
        "id": "11f0249ebb4889c62f9e1a3f7ea2f24d0a9fb743bef24a7bc689480b4644b221",
        "domain": "omni",
        "question": "How many positive integers $n$ with $n \\leq 100$ can be expressed as the sum of four or more consecutive positive integers?",
        "answer": "63",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "fermat",
        "source_index": 2872,
        "difficulty": 2.5
      },
      {
        "id": "1f175256ce7797f754f23401a7445af0154d03abad31d1bcfa7858f894fe161c",
        "domain": "omni",
        "question": "Unit circle $\\Omega$ has points $X, Y, Z$ on its circumference so that $X Y Z$ is an equilateral triangle. Let $W$ be a point other than $X$ in the plane such that triangle $W Y Z$ is also equilateral. Determine the area of the region inside triangle $W Y Z$ that lies outside circle $\\Omega$.",
        "answer": "$\\frac{3 \\sqrt{3}-\\pi}{3}$",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_11",
        "source_index": 1939,
        "difficulty": 4.5
      },
      {
        "id": "ec9fc7fe62da0289bbca9fce69fe40af7464e2a56beaccf3d0ac5c6bef1489b7",
        "domain": "omni",
        "question": "Find the sum of the series $\\sum_{n=1}^{\\infty} \\frac{1}{n^{2}+2n}$.",
        "answer": "\\frac{3}{4}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Calculus -> Integral Calculus -> Series -> Other",
        "source": "HMMT_2",
        "source_index": 1186,
        "difficulty": 5.0
      },
      {
        "id": "e9d85cf3f3cc1e58291ec909858b86b03223a3715ba65da1c64b76b7270d14d9",
        "domain": "omni",
        "question": "Let $a_{1}=3$, and for $n>1$, let $a_{n}$ be the largest real number such that $$4\\left(a_{n-1}^{2}+a_{n}^{2}\\right)=10 a_{n-1} a_{n}-9$$ What is the largest positive integer less than $a_{8}$ ?",
        "answer": "335",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Recurrence Relations -> Other",
        "source": "HMMT_2",
        "source_index": 2576,
        "difficulty": 5.0
      },
      {
        "id": "549bbd0f75f61a0a663ec175229b844d1ce8da6c3e97a7044531a0cea4d70842",
        "domain": "omni",
        "question": "Sherry and Val are playing a game. Sherry has a deck containing 2011 red cards and 2012 black cards, shuffled randomly. Sherry flips these cards over one at a time, and before she flips each card over, Val guesses whether it is red or black. If Val guesses correctly, she wins 1 dollar; otherwise, she loses 1 dollar. In addition, Val must guess red exactly 2011 times. If Val plays optimally, what is her expected profit from this game?",
        "answer": "\\frac{1}{4023}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 609,
        "difficulty": 5.25
      },
      {
        "id": "e64e884011253a098c40457291bb16003997ba557233bb65dfc6b51c9c35ecdc",
        "domain": "omni",
        "question": "Let $E$ be a three-dimensional ellipsoid. For a plane $p$, let $E(p)$ be the projection of $E$ onto the plane $p$. The minimum and maximum areas of $E(p)$ are $9 \\pi$ and $25 \\pi$, and there exists a $p$ where $E(p)$ is a circle of area $16 \\pi$. If $V$ is the volume of $E$, compute $V / \\pi$.",
        "answer": "75",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> Volume",
        "source": "HMMT_2",
        "source_index": 936,
        "difficulty": 4.5
      },
      {
        "id": "246b037217fcb17d8aaf52f368ad35af1517d2532f47902615c5139190f62e4d",
        "domain": "omni",
        "question": "Let $n$ be the 200th smallest positive real solution to the equation $x-\\frac{\\pi}{2}=\\tan x$. Find the greatest integer that does not exceed $\\frac{n}{2}$.",
        "answer": "314",
        "topic": "Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2233,
        "difficulty": 6.0
      },
      {
        "id": "896f030fc5aa3a806519b04293d564f10e79ca67833a4cf8b8bb4ae566388a28",
        "domain": "omni",
        "question": "Find the largest real number $c$ such that $$\\sum_{i=1}^{101} x_{i}^{2} \\geq c M^{2}$$ whenever $x_{1}, \\ldots, x_{101}$ are real numbers such that $x_{1}+\\cdots+x_{101}=0$ and $M$ is the median of $x_{1}, \\ldots, x_{101}$.",
        "answer": "\\frac{5151}{50}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Applied Mathematics -> Statistics -> Mathematical Statistics",
        "source": "HMMT_2",
        "source_index": 1370,
        "difficulty": 4.5
      },
      {
        "id": "2efa530a923db464158c2b693d1a6fd130c272f73415f5992e3416f4425a45cd",
        "domain": "omni",
        "question": "Find the largest integer that divides $m^{5}-5 m^{3}+4 m$ for all $m \\geq 5$.",
        "answer": "120",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 505,
        "difficulty": 5.25
      },
      {
        "id": "cc70f958e7f54c566cb4e5d52352de8e23fe90f40ef2d422e759ad5c3b0dc411",
        "domain": "omni",
        "question": "There are $2018$ players sitting around a round table. At the beginning of the game we arbitrarily deal all the cards from a deck of $K$ cards to the players (some players may receive no cards). In each turn we choose a player who draws one card from each of the two neighbors. It is only allowed to choose a player whose each neighbor holds a nonzero number of cards. The game terminates when there is no such player. Determine the largest possible value of $K$ such that, no matter how we deal the cards and how we choose the players, the game always terminates after a finite number of turns.",
        "answer": "2017",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "czech-polish-slovak matches",
        "source_index": 3558,
        "difficulty": 7.0
      },
      {
        "id": "3ce98c3b7d65a2c0c622c69d96760ecc58b4d3c6afb18137840b20379c6f6ff9",
        "domain": "omni",
        "question": "You want to arrange the numbers $1,2,3, \\ldots, 25$ in a sequence with the following property: if $n$ is divisible by $m$, then the $n$th number is divisible by the $m$ th number. How many such sequences are there?",
        "answer": "24",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 672,
        "difficulty": 5.0
      },
      {
        "id": "cfeacb8bb5b795d405c97fec28a9543400fc36012c86fbdabb58c61c128b7689",
        "domain": "omni",
        "question": "Let $\\triangle A B C$ be a scalene triangle. Let $h_{a}$ be the locus of points $P$ such that $|P B-P C|=|A B-A C|$. Let $h_{b}$ be the locus of points $P$ such that $|P C-P A|=|B C-B A|$. Let $h_{c}$ be the locus of points $P$ such that $|P A-P B|=|C A-C B|$. In how many points do all of $h_{a}, h_{b}$, and $h_{c}$ concur?",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Conic Sections -> Other",
        "source": "HMMT_2",
        "source_index": 2397,
        "difficulty": 5.0
      },
      {
        "id": "3fafe46e699e35d5cd4b6a6de8b65548b97211fffbbf31f28559a640042ebdd3",
        "domain": "omni",
        "question": "For how many $n$ with $1 \\leq n \\leq 100$ can a unit square be divided into $n$ congruent figures?",
        "answer": "100",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1969,
        "difficulty": 3.5
      },
      {
        "id": "73391c37192a5bc373edf5a16335f62ff3b8731b0ce030a99a05cd471eee1355",
        "domain": "omni",
        "question": "A set of six edges of a regular octahedron is called Hamiltonian cycle if the edges in some order constitute a single continuous loop that visits each vertex exactly once. How many ways are there to partition the twelve edges into two Hamiltonian cycles?",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1408,
        "difficulty": 4.0
      },
      {
        "id": "c0163441153df7462c706e8ee8e080381a72d913f1ca55215e548464e3575b27",
        "domain": "omni",
        "question": "Boris was given a Connect Four game set for his birthday, but his color-blindness makes it hard to play the game. Still, he enjoys the shapes he can make by dropping checkers into the set. If the number of shapes possible modulo (horizontal) flips about the vertical axis of symmetry is expressed as $9(1+2+\\cdots+n)$, find $n$.",
        "answer": "729",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 966,
        "difficulty": 5.0
      },
      {
        "id": "f13e3c362eb03ca025b3d877e6553be8497f0cd0a6773c4ad8d762f7434da093",
        "domain": "omni",
        "question": "The game of rock-scissors is played just like rock-paper-scissors, except that neither player is allowed to play paper. You play against a poorly-designed computer program that plays rock with $50 \\%$ probability and scissors with $50 \\%$ probability. If you play optimally against the computer, find the probability that after 8 games you have won at least 4.",
        "answer": "\\frac{163}{256}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2545,
        "difficulty": 4.0
      },
      {
        "id": "b453c99de6fa36563edcd2673938616a013ed75aad434207930c31207b3831fd",
        "domain": "omni",
        "question": "A circle inscribed in a square has two chords as shown in a pair. It has radius 2, and $P$ bisects $T U$. The chords' intersection is where? Answer the question by giving the distance of the point of intersection from the center of the circle.",
        "answer": "2\\sqrt{2} - 2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 3356,
        "difficulty": 3.5
      },
      {
        "id": "22f9570561c3b59846670eacb7a581d5aa9e4f1c221c346220a274c0903e85d3",
        "domain": "omni",
        "question": "A jar contains 97 marbles that are either red, green, or blue. Neil draws two marbles from the jar without replacement and notes that the probability that they would be the same color is $\\frac{5}{12}$. After Neil puts his marbles back, Jerry draws two marbles from the jar with replacement. Compute the probability that the marbles that Jerry draws are the same color.",
        "answer": "\\frac{41}{97}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1785,
        "difficulty": 5.25
      },
      {
        "id": "2d5e6a377c06921b7d746402277d3d462d84b5de5050270c6b064ea4e91c5e92",
        "domain": "omni",
        "question": "On a computer screen is the single character a. The computer has two keys: c (copy) and p (paste), which may be pressed in any sequence. Pressing p increases the number of a's on screen by the number that were there the last time c was pressed. c doesn't change the number of a's on screen. Determine the fewest number of keystrokes required to attain at least 2018 a's on screen. (Note: pressing p before the first press of c does nothing).",
        "answer": "21",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1903,
        "difficulty": 5.0
      },
      {
        "id": "48670716d0f62cf1597672587b79202f9bed397fa4864d4056534a7d4bf52035",
        "domain": "omni",
        "question": "A standard $n$-sided die has $n$ sides labeled 1 to $n$. Luis, Luke, and Sean play a game in which they roll a fair standard 4-sided die, a fair standard 6-sided die, and a fair standard 8-sided die, respectively. They lose the game if Luis's roll is less than Luke's roll, and Luke's roll is less than Sean's roll. Compute the probability that they lose the game.",
        "answer": "\\frac{1}{4}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1625,
        "difficulty": 3.5
      },
      {
        "id": "6cac629240d17a3fd7eab0e21ce726b1cc32dbba719d5304cf3fcef023e4c1e6",
        "domain": "omni",
        "question": "$M$ is an $8 \\times 8$ matrix. For $1 \\leq i \\leq 8$, all entries in row $i$ are at least $i$, and all entries on column $i$ are at least $i$. What is the minimum possible sum of the entries of $M$ ?",
        "answer": "372",
        "topic": "Mathematics -> Algebra -> Linear Algebra -> Matrices",
        "source": "HMMT_2",
        "source_index": 738,
        "difficulty": 4.5
      },
      {
        "id": "4edf9eb2c0782c810572062e1b896040b8d5ec53ee23fe5bbcdc3529aea24313",
        "domain": "omni",
        "question": "A total of $n$ points are equally spaced around a circle and are labelled with the integers 1 to $n$, in order. Two points are called diametrically opposite if the line segment joining them is a diameter of the circle. If the points labelled 7 and 35 are diametrically opposite, then what is the value of $n$?",
        "answer": "56",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "fermat",
        "source_index": 2714,
        "difficulty": 2.5
      },
      {
        "id": "81d369e209edfda01c8fb9841fccaa3e6ae29262d03721f18969665ab8fbf975",
        "domain": "omni",
        "question": "Consider the system \\begin{align*}x + y  &= z + u,\\\\2xy & = zu.\\end{align*} Find the greatest value of the real constant $m$ such that $m \\leq x/y$ for any positive integer solution $(x,y,z,u)$ of the system, with $x \\geq y$.",
        "answer": "3 + 2\\sqrt{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Other",
        "source": "imo_shortlist",
        "source_index": 3908,
        "difficulty": 7.0
      },
      {
        "id": "f59905af0718f499b3b05378bad51e16052d11a46782b98605a6e073ec53ed16",
        "domain": "omni",
        "question": "A loonie is a $\\$ 1$ coin and a dime is a $\\$ 0.10$ coin. One loonie has the same mass as 4 dimes. A bag of dimes has the same mass as a bag of loonies. The coins in the bag of loonies are worth $\\$ 400$ in total. How much are the coins in the bag of dimes worth?",
        "answer": "160",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3400,
        "difficulty": 2.0
      },
      {
        "id": "40a6c88f6764a21e2c0dc38e726e7c5a543070cf81aaddf711340d341a0aa246",
        "domain": "omni",
        "question": "Each unit square of a $4 \\times 4$ square grid is colored either red, green, or blue. Over all possible colorings of the grid, what is the maximum possible number of L-trominos that contain exactly one square of each color?",
        "answer": "18",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1131,
        "difficulty": 5.0
      },
      {
        "id": "4f8df238501ef9411bae0a46ff1e2f8ac55380a230abee122cdaa7118058e300",
        "domain": "omni",
        "question": "Compute the positive integer less than 1000 which has exactly 29 positive proper divisors.",
        "answer": "720",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 562,
        "difficulty": 4.0
      },
      {
        "id": "0f776550f16d95194b1fd418699809385bb6bec9152740b04ba6bd9b5a6e21a2",
        "domain": "omni",
        "question": "The product of the roots of the equation \\((x-4)(x-2)+(x-2)(x-6)=0\\) is",
        "answer": "10",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "fermat",
        "source_index": 3403,
        "difficulty": 2.5
      },
      {
        "id": "dabff91cb7215fd0f65bdc6653a740942c1c468dab4477956d244c702b9c812a",
        "domain": "omni",
        "question": "Let $a$ be a positive integer such that $2a$ has units digit 4. What is the sum of the possible units digits of $3a$?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2641,
        "difficulty": 3.5
      },
      {
        "id": "fe34644d6e000167e2144e0544e517af34a17bb5850fc12c4ec0edb9f1075521",
        "domain": "omni",
        "question": "Three tanks contain water. The number of litres in each is shown in the table: Tank A: 3600 L, Tank B: 1600 L, Tank C: 3800 L. Water is moved from each of Tank A and Tank C into Tank B so that each tank contains the same volume of water. How many litres of water are moved from Tank A to Tank B?",
        "answer": "600",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2848,
        "difficulty": 2.5
      },
      {
        "id": "f26825f46ae4307680fc619c58291bf973357d65788192affce14a3d4e620f58",
        "domain": "omni",
        "question": "The value of \\( \\frac{1}{2} + \\frac{2}{4} + \\frac{4}{8} + \\frac{8}{16} \\) is what?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "pascal",
        "source_index": 3543,
        "difficulty": 1.0
      },
      {
        "id": "e98242bcc1fa9a9b82adc607e955834294aef16905b9de6f72aa9a1d350b7d3f",
        "domain": "omni",
        "question": "For each positive integer $k$, let $A(k)$ be the number of odd divisors of $k$ in the interval $[1, \\sqrt{2k})$. Evaluate\n\\[\n\\sum_{k=1}^\\infty (-1)^{k-1} \\frac{A(k)}{k}.\n\\]",
        "answer": "\\frac{\\pi^2}{16}",
        "topic": "Mathematics -> Number Theory -> Divisors -> Other; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Other; Mathematics -> Algebra -> Complex Numbers -> Other",
        "source": "putnam",
        "source_index": 3171,
        "difficulty": 9.0
      },
      {
        "id": "f2de9ec439e5ce4df55c11bc6ac89575e0b7a32f92b22ffc66bb382aed3e0d52",
        "domain": "omni",
        "question": "Suppose $a, b$ and $c$ are integers such that the greatest common divisor of $x^{2}+a x+b$ and $x^{2}+b x+c$ is $x+1$ (in the ring of polynomials in $x$ with integer coefficients), and the least common multiple of $x^{2}+a x+b$ and $x^{2}+b x+c$ is $x^{3}-4 x^{2}+x+6$. Find $a+b+c$.",
        "answer": "-6",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1239,
        "difficulty": 5.0
      },
      {
        "id": "ef3bcd7a4673d4f69fc9163b57c806df50be7da9e833e44afb50d00ca23ce5e6",
        "domain": "omni",
        "question": "Each of the following 15 cards has a letter on one side and a positive integer on the other side. What is the minimum number of cards that need to be turned over to check if the following statement is true? 'If a card has a lower case letter on one side, then it has an odd integer on the other side.'",
        "answer": "3",
        "topic": "Mathematics -> Discrete Mathematics -> Logic",
        "source": "pascal",
        "source_index": 3021,
        "difficulty": 2.0
      },
      {
        "id": "b0e3557fc288545ec3acf23b2f9857af1b8ba8a8d0cd20990cb29f4b2a6c4537",
        "domain": "omni",
        "question": "Compute the number of ordered pairs of integers $(x, y)$ such that $x^{2}+y^{2}<2019$ and $$x^{2}+\\min (x, y)=y^{2}+\\max (x, y)$$",
        "answer": "127",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2210,
        "difficulty": 5.0
      },
      {
        "id": "169688a0424d77f937117d4a1550396bb43529c35b7b8046e3b3e4962a1ead87",
        "domain": "omni",
        "question": "Compute the number of positive real numbers $x$ that satisfy $\\left(3 \\cdot 2^{\\left\\lfloor\\log _{2} x\\right\\rfloor}-x\\right)^{16}=2022 x^{13}$.",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1418,
        "difficulty": 5.0
      },
      {
        "id": "480a3cd8a73f76d3f98ecd22f37d8a2630d37417b5511b48947388ff7b74fa6f",
        "domain": "omni",
        "question": "Two real numbers $x$ and $y$ are such that $8 y^{4}+4 x^{2} y^{2}+4 x y^{2}+2 x^{3}+2 y^{2}+2 x=x^{2}+1$. Find all possible values of $x+2 y^{2}$.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1544,
        "difficulty": 5.0
      },
      {
        "id": "cbfd7d5b3873e4af1f71a03b5d7efa38637e30929f3b531829114f04e57ab6b3",
        "domain": "omni",
        "question": "Over all real numbers $x$ and $y$, find the minimum possible value of $$ (x y)^{2}+(x+7)^{2}+(2 y+7)^{2} $$",
        "answer": "45",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2139,
        "difficulty": 5.25
      },
      {
        "id": "e228a1b5c554ebffa9813ce6e455e3ac7f9c7eb5d9c01a4ecec0bb85d96b0f49",
        "domain": "omni",
        "question": "The symbol $\\diamond$ is defined so that $a \\diamond b=\\frac{a+b}{a \\times b}$. What is the value of $3 \\diamond 6$?",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2677,
        "difficulty": 1.0
      },
      {
        "id": "299f86c68f35d9fde0d49f3d5d2758ca532342e54b1c70cf2a9714b2720273f9",
        "domain": "omni",
        "question": "Knot is on an epic quest to save the land of Hyruler from the evil Gammadorf. To do this, he must collect the two pieces of the Lineforce, then go to the Temple of Lime. As shown on the figure, Knot starts on point $K$, and must travel to point $T$, where $O K=2$ and $O T=4$. However, he must first reach both solid lines in the figure below to collect the pieces of the Lineforce. What is the minimal distance Knot must travel to do so?",
        "answer": "2 \\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 822,
        "difficulty": 5.0
      },
      {
        "id": "2958117f0f20608f7a0db8f7ded652a0c9f1da55ee2256f0bcb41ea6e17fcad6",
        "domain": "omni",
        "question": "A number is chosen uniformly at random from the set of all positive integers with at least two digits, none of which are repeated. Find the probability that the number is even.",
        "answer": "\\frac{41}{81}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2303,
        "difficulty": 4.5
      },
      {
        "id": "c63a9e790363a8d266eebc054c59ec09ef197e7acbc86e6426bba50b5db7bb66",
        "domain": "omni",
        "question": "Lily and Sarah are playing a game. They each choose a real number at random between -1 and 1. They then add the squares of their numbers together. If the result is greater than or equal to 1, Lily wins, and if the result is less than 1, Sarah wins. What is the probability that Sarah wins?",
        "answer": "\\frac{\\pi}{4}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1680,
        "difficulty": 3.5
      },
      {
        "id": "8d8020bc11f778f4a3856aaab1a45a2e425196cc9769a184e19c2f2465ca2126",
        "domain": "omni",
        "question": "Suppose $a_{1}, a_{2}, \\ldots, a_{100}$ are positive real numbers such that $$a_{k}=\\frac{k a_{k-1}}{a_{k-1}-(k-1)}$$ for $k=2,3, \\ldots, 100$. Given that $a_{20}=a_{23}$, compute $a_{100}$.",
        "answer": "215",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1327,
        "difficulty": 5.0
      },
      {
        "id": "526e5059e83e427d8809844db9766d35507364e2de1e16cf33d38eea24f4814d",
        "domain": "omni",
        "question": "Consider $9$ points in space, no four of which are coplanar. Each pair of points is joined by an edge (that is, a line segment) and each edge is either colored blue or red or left uncolored. Find the smallest value of  $\\,n\\,$ such that whenever exactly $\\,n\\,$ edges are colored, the set of colored edges necessarily contains a triangle all of whose edges have the same color.",
        "answer": "33",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "imo",
        "source_index": 3902,
        "difficulty": 9.0
      },
      {
        "id": "a03009ff525a964ac61108ee228b50baef39c71fca4cda23fe770779c03b2815",
        "domain": "omni",
        "question": "Define the sequence \\left\\{x_{i}\\right\\}_{i \\geq 0} by $x_{0}=x_{1}=x_{2}=1$ and $x_{k}=\\frac{x_{k-1}+x_{k-2}+1}{x_{k-3}}$ for $k>2$. Find $x_{2013}$.",
        "answer": "9",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series",
        "source": "HMMT_11",
        "source_index": 1975,
        "difficulty": 4.5
      },
      {
        "id": "24c973bfc94efb77b15a308390b5d5f0e3b1fea0c0e2ad547a2d535603b92edb",
        "domain": "omni",
        "question": "The minute hand on a clock points at the 12. After rotating $120^{\\circ}$ clockwise, which number will it point at?",
        "answer": "4",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "cayley",
        "source_index": 3394,
        "difficulty": 1.5
      },
      {
        "id": "49fe2934905dbfe9b4f09ef1a398b3cc24509cfff93d4d378b94453eb67ff8fe",
        "domain": "omni",
        "question": "Ava and Tiffany participate in a knockout tournament consisting of a total of 32 players. In each of 5 rounds, the remaining players are paired uniformly at random. In each pair, both players are equally likely to win, and the loser is knocked out of the tournament. The probability that Ava and Tiffany play each other during the tournament is $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "116",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1204,
        "difficulty": 4.0
      },
      {
        "id": "1b20f3205d269ae6b3c4cc0eef5731461c4b3f4eed581fce32ea779835147250",
        "domain": "omni",
        "question": "Allen and Brian are playing a game in which they roll a 6-sided die until one of them wins. Allen wins if two consecutive rolls are equal and at most 3. Brian wins if two consecutive rolls add up to 7 and the latter is at most 3. What is the probability that Allen wins?",
        "answer": "\\frac{5}{12}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 1998,
        "difficulty": 4.5
      },
      {
        "id": "5e6d775776311c8645eeca730d8367b2bf366915c6e7ff06e7c8858a7475b7ea",
        "domain": "omni",
        "question": "Find the largest number $n$ such that $(2004!)!$ is divisible by $((n!)!)!$.",
        "answer": "6",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Prealgebra -> Factorials -> Other",
        "source": "HMMT_2",
        "source_index": 1411,
        "difficulty": 5.0
      },
      {
        "id": "d657608289b7bc16086091dc64edc42ccb311f58706c8d321af6e6c9e571f18a",
        "domain": "omni",
        "question": "Find the total number of occurrences of the digits $0,1 \\ldots, 9$ in the entire guts round. If your answer is $X$ and the actual value is $Y$, your score will be $\\max \\left(0,20-\\frac{|X-Y|}{2}\\right)$",
        "answer": "559",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1866,
        "difficulty": 5.25
      },
      {
        "id": "f0a7979b9b097523a3869e81bc4a9ba485723e0037b2aad4d4db728039fa35ad",
        "domain": "omni",
        "question": "For each positive integer $n$ and non-negative integer $k$, define $W(n, k)$ recursively by $$ W(n, k)= \\begin{cases}n^{n} & k=0 \\\\ W(W(n, k-1), k-1) & k>0\\end{cases} $$ Find the last three digits in the decimal representation of $W(555,2)$.",
        "answer": "875",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1051,
        "difficulty": 5.0
      },
      {
        "id": "e2c0be456b7e8809c5823e1bf0e50b8ddd27cd3abf9face8db98bf542d680aeb",
        "domain": "omni",
        "question": "$A B C D$ is a cyclic quadrilateral in which $A B=3, B C=5, C D=6$, and $A D=10 . M, I$, and $T$ are the feet of the perpendiculars from $D$ to lines $A B, A C$, and $B C$ respectively. Determine the value of $M I / I T$.",
        "answer": "\\frac{25}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "HMMT_2",
        "source_index": 1505,
        "difficulty": 5.0
      },
      {
        "id": "39f7aa7d3a692de5d69fd6d569b4f4b73da637144eefdac7ee860d9450adde5c",
        "domain": "omni",
        "question": "Determine the positive real value of $x$ for which $$\\sqrt{2+A C+2 C x}+\\sqrt{A C-2+2 A x}=\\sqrt{2(A+C) x+2 A C}$$",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2051,
        "difficulty": 5.0
      },
      {
        "id": "d9a6288256e138a4daccf915f78a8e3fc1907b207b157e6adfb784f242ad7ebd",
        "domain": "omni",
        "question": "There are three pairs of real numbers \\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right), and \\left(x_{3}, y_{3}\\right) that satisfy both $x^{3}-3 x y^{2}=2005$ and $y^{3}-3 x^{2} y=2004$. Compute \\left(1-\\frac{x_{1}}{y_{1}}\\right)\\left(1-\\frac{x_{2}}{y_{2}}\\right)\\left(1-\\frac{x_{3}}{y_{3}}\\right).",
        "answer": "1/1002",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1142,
        "difficulty": 5.0
      },
      {
        "id": "cba88d389d88ed1b0822d4014069b69091c4c4db96656783971ef575ca6c51d3",
        "domain": "omni",
        "question": "Urn A contains 4 white balls and 2 red balls. Urn B contains 3 red balls and 3 black balls. An urn is randomly selected, and then a ball inside of that urn is removed. We then repeat the process of selecting an urn and drawing out a ball, without returning the first ball. What is the probability that the first ball drawn was red, given that the second ball drawn was black?",
        "answer": "7/15",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Applied Mathematics -> Statistics -> Mathematical Statistics",
        "source": "HMMT_2",
        "source_index": 1193,
        "difficulty": 3.5
      },
      {
        "id": "a66d97ca5a38a2896c228d3c8707bb58488f8118d448d7bed825041d33c4ae8b",
        "domain": "omni",
        "question": "Calvin has a bag containing 50 red balls, 50 blue balls, and 30 yellow balls. Given that after pulling out 65 balls at random (without replacement), he has pulled out 5 more red balls than blue balls, what is the probability that the next ball he pulls out is red?",
        "answer": "\\frac{9}{26}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1133,
        "difficulty": 4.0
      },
      {
        "id": "57bde6a7323a8be609d221af89776a147a5ff6ad8914d03c13b93cfd1a7a759d",
        "domain": "omni",
        "question": "Compute the number of ways to color 3 cells in a $3 \\times 3$ grid so that no two colored cells share an edge.",
        "answer": "22",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1683,
        "difficulty": 4.0
      },
      {
        "id": "f1a80cac830f105ea080a7290babf87b159c01ad71480f88efcf6c9d6b24d34d",
        "domain": "omni",
        "question": "Sixteen wooden Cs are placed in a 4-by-4 grid, all with the same orientation, and each is to be colored either red or blue. A quadrant operation on the grid consists of choosing one of the four two-by-two subgrids of Cs found at the corners of the grid and moving each C in the subgrid to the adjacent square in the subgrid that is 90 degrees away in the clockwise direction, without changing the orientation of the C. Given that two colorings are the considered same if and only if one can be obtained from the other by a series of quadrant operations, determine the number of distinct colorings of the Cs.",
        "answer": "1296",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2031,
        "difficulty": 4.5
      },
      {
        "id": "80ea44681e67691595fae574734fe2a80c8914bab343de5d3cf8b7f0d9df3762",
        "domain": "omni",
        "question": "Pick a random integer between 0 and 4095, inclusive. Write it in base 2 (without any leading zeroes). What is the expected number of consecutive digits that are not the same (that is, the expected number of occurrences of either 01 or 10 in the base 2 representation)?",
        "answer": "\\frac{20481}{4096}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 2350,
        "difficulty": 5.0
      },
      {
        "id": "07bb83fa61e47eed63be9d98507b89754532ff6c80232cc5a9cd5e55547e8e1d",
        "domain": "omni",
        "question": "Find the expected value of the number formed by rolling a fair 6-sided die with faces numbered 1, 2, 3, 5, 7, 9 infinitely many times.",
        "answer": "\\frac{1}{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2381,
        "difficulty": 4.0
      },
      {
        "id": "fd31b3242848f5443badfbf749de5b74324fb64dc354e8ab7d2a8d8ae110b0e3",
        "domain": "omni",
        "question": "Amy and Ben need to eat 1000 total carrots and 1000 total muffins. The muffins can not be eaten until all the carrots are eaten. Furthermore, Amy can not eat a muffin within 5 minutes of eating a carrot and neither can Ben. If Amy eats 40 carrots per minute and 70 muffins per minute and Ben eats 60 carrots per minute and 30 muffins per minute, what is the minimum number of minutes it will take them to finish the food?",
        "answer": "23.5",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 758,
        "difficulty": 5.25
      },
      {
        "id": "9c5e1fdd10a4b565eae10feb0386dc96f22b36b48302ec21ed78deb6aa2daf72",
        "domain": "omni",
        "question": "Jessica has three marbles colored red, green, and blue. She randomly selects a non-empty subset of them (such that each subset is equally likely) and puts them in a bag. You then draw three marbles from the bag with replacement. The colors you see are red, blue, red. What is the probability that the only marbles in the bag are red and blue?",
        "answer": "\\frac{27}{35}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 2530,
        "difficulty": 5.25
      },
      {
        "id": "d308e640b361a216826ec9df5a53610a2a032ab8d8b8faf3e975e3683cb8a89f",
        "domain": "omni",
        "question": "A student at Harvard named Kevin was counting his stones by 11. He messed up $n$ times and instead counted 9s and wound up at 2007. How many values of $n$ could make this limerick true?",
        "answer": "21",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 1689,
        "difficulty": 4.0
      },
      {
        "id": "cbb53d76fb8772bba1927823b267d93e209b3f07ababf5333bbd1b340a8b59b5",
        "domain": "omni",
        "question": "Let $ a_1 \\equal{} 11^{11}, \\, a_2 \\equal{} 12^{12}, \\, a_3 \\equal{} 13^{13}$, and $ a_n \\equal{} |a_{n \\minus{} 1} \\minus{} a_{n \\minus{} 2}| \\plus{} |a_{n \\minus{} 2} \\minus{} a_{n \\minus{} 3}|, n \\geq 4.$ Determine $ a_{14^{14}}$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Other; Mathematics -> Number Theory -> Other",
        "source": "imo_shortlist",
        "source_index": 4041,
        "difficulty": 9.5
      },
      {
        "id": "22515ee7bd7f71ba6ea7a4c055c829feacde77bbea2571e1971b58435e4d2c1d",
        "domain": "omni",
        "question": "Let $S$ be a subset of the set $\\{1,2,3, \\ldots, 2015\\}$ such that for any two elements $a, b \\in S$, the difference $a-b$ does not divide the sum $a+b$. Find the maximum possible size of $S$.",
        "answer": "672",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2219,
        "difficulty": 4.0
      },
      {
        "id": "8597ba1433759c3376929236c8eee4018ff3d7ddd24cccf692671d9583b473e6",
        "domain": "omni",
        "question": "Meghana writes two (not necessarily distinct) primes $q$ and $r$ in base 10 next to each other on a blackboard, resulting in the concatenation of $q$ and $r$ (for example, if $q=13$ and $r=5$, the number on the blackboard is now 135). She notices that three more than the resulting number is the square of a prime $p$. Find all possible values of $p$.",
        "answer": "5",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2161,
        "difficulty": 4.0
      },
      {
        "id": "84ade5d46bffbd6d9a194fa35b507d3100444b59b8f5a7a689b525ef77ba27d0",
        "domain": "omni",
        "question": "If $x=1$ is a solution of the equation $x^{2} + ax + 1 = 0$, what is the value of $a$?",
        "answer": "-2",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "fermat",
        "source_index": 2917,
        "difficulty": 2.5
      },
      {
        "id": "c8b66bf703b7539ad517070e25a95e11ef3cb5a11487bbc9b0b0a42b44cf626c",
        "domain": "omni",
        "question": "Find the number of 10-digit numbers $\\overline{a_{1} a_{2} \\cdots a_{10}}$ which are multiples of 11 such that the digits are non-increasing from left to right, i.e. $a_{i} \\geq a_{i+1}$ for each $1 \\leq i \\leq 9$.",
        "answer": "2001",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Congruences; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2465,
        "difficulty": 5.5
      },
      {
        "id": "085ecd1c10bfd5063bb425fd5e0a572c282796b1a5e09ffbed26fd421acf0ce1",
        "domain": "omni",
        "question": "A chord is drawn on a circle by choosing two points uniformly at random along its circumference. This is done two more times to obtain three total random chords. The circle is cut along these three lines, splitting it into pieces. The probability that one of the pieces is a triangle is $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
        "answer": "115",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2527,
        "difficulty": 4.0
      },
      {
        "id": "23fe85cfaa55c3309671505e19c880812d6ae5b02ff82d4e4da8ceb49d87f0d2",
        "domain": "omni",
        "question": "Define $a$ ? $=(a-1) /(a+1)$ for $a \\neq-1$. Determine all real values $N$ for which $(N ?)$ ?=\\tan 15.",
        "answer": "-2-\\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 947,
        "difficulty": 4.0
      },
      {
        "id": "0477d47820f10a6123d520719b0cf0543dae7ac840fd38b63bec7e19b46da094",
        "domain": "omni",
        "question": "Let $N$ be a three-digit integer such that the difference between any two positive integer factors of $N$ is divisible by 3 . Let $d(N)$ denote the number of positive integers which divide $N$. Find the maximum possible value of $N \\cdot d(N)$.",
        "answer": "5586",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 778,
        "difficulty": 5.0
      },
      {
        "id": "5ade19c440781cd173b21adb500dd728f1a576178ccd9a2d49e39ea02e8f83fc",
        "domain": "omni",
        "question": "For which maximal $N$ there exists an $N$-digit number with the following property: among any sequence of its consecutive decimal digits some digit is present once only?\n\t\n\tAlexey Glebov",
        "answer": "1023",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "problems_from_the_kvant_magazine",
        "source_index": 4301,
        "difficulty": 6.0
      },
      {
        "id": "e57c7c673ddef29349ac8e49e8b713137a9c70c0bf28f47c0f87e87c1a5df65d",
        "domain": "omni",
        "question": "Regular polygons $I C A O, V E N T I$, and $A L B E D O$ lie on a plane. Given that $I N=1$, compute the number of possible values of $O N$.",
        "answer": "2",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 1415,
        "difficulty": 5.0
      },
      {
        "id": "670b1e433686a5467bb10fae0c4c8601609c13d8815e7d0f82cd83ad3a71a0a3",
        "domain": "omni",
        "question": "I have written a strictly increasing sequence of six positive integers, such that each number (besides the first) is a multiple of the one before it, and the sum of all six numbers is 79 . What is the largest number in my sequence?",
        "answer": "48",
        "topic": "Mathematics -> Algebra -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 842,
        "difficulty": 3.5
      },
      {
        "id": "15e7fa50f63d563606e93399713b1e3c1a1e8e7754f69a67f561d051dc015c63",
        "domain": "omni",
        "question": "Compute the sum of all positive integers $a \\leq 26$ for which there exist integers $b$ and $c$ such that $a+23 b+15 c-2$ and $2 a+5 b+14 c-8$ are both multiples of 26.",
        "answer": "31",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2412,
        "difficulty": 4.0
      },
      {
        "id": "f8e8b1ccaee9b9eaedd3a7e2a4e3a072adee252b1598b431782857cf6fa0f316",
        "domain": "omni",
        "question": "Let $a_{1}, a_{2}, a_{3}, a_{4}, a_{5}$ be real numbers whose sum is 20. Determine with proof the smallest possible value of \\(\\sum_{1 \\leq i<j \\leq 5}\\left\\lfloor a_{i}+a_{j}\\right\\rfloor\\).",
        "answer": "\\[\n72\n\\]",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Other",
        "source": "HMMT_2",
        "source_index": 371,
        "difficulty": 4.5
      },
      {
        "id": "e6721a23b62316ca8d0c724b5d1dc5d2d95c91d966f418544b69de421ea999fa",
        "domain": "omni",
        "question": "Given a acute triangle $PA_1B_1$ is inscribed in the circle $\\Gamma$ with radius $1$. for all  integers $n \\ge 1$ are defined:\n$C_n$ the foot of the perpendicular from $P$ to $A_nB_n$\n$O_n$ is the center of $\\odot (PA_nB_n)$\n$A_{n+1}$ is the foot of the perpendicular from $C_n$ to $PA_n$\n$B_{n+1} \\equiv PB_n \\cap O_nA_{n+1}$\n\nIf $PC_1 =\\sqrt{2}$, find the length of $PO_{2015}$ \n\n[hide=Source]Cono Sur Olympiad - 2015 - Day 1 - Problem 3[/hide]",
        "answer": "\\frac{1}{2^{1007}}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "cono_sur_olympiad",
        "source_index": 3671,
        "difficulty": 8.0
      },
      {
        "id": "4225e008fedb7114560cf9506fd86a87b42f7447e6b1ef1153bab58268e75667",
        "domain": "omni",
        "question": "Let $s(n)$ denote the sum of the digits (in base ten) of a positive integer $n$. Compute the number of positive integers $n$ at most $10^{4}$ that satisfy $$s(11 n)=2 s(n)$$",
        "answer": "2530",
        "topic": "Mathematics -> Number Theory -> Digits and Sums -> Other; Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1821,
        "difficulty": 4.5
      },
      {
        "id": "8d96c02e9454273cbe11a697a262a51991199a7fe66276c7dc5cb6dc0b421087",
        "domain": "omni",
        "question": "To set up for a Fourth of July party, David is making a string of red, white, and blue balloons. He places them according to the following rules: - No red balloon is adjacent to another red balloon. - White balloons appear in groups of exactly two, and groups of white balloons are separated by at least two non-white balloons. - Blue balloons appear in groups of exactly three, and groups of blue balloons are separated by at least three non-blue balloons. If David uses over 600 balloons, determine the smallest number of red balloons that he can use.",
        "answer": "99",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 2328,
        "difficulty": 5.25
      },
      {
        "id": "266b733b5c21aee20f81a77a57af703b5cea9f33c63e9f098a8119dfda817526",
        "domain": "omni",
        "question": "Let $P_{1}, P_{2}, \\ldots, P_{8}$ be 8 distinct points on a circle. Determine the number of possible configurations made by drawing a set of line segments connecting pairs of these 8 points, such that: (1) each $P_{i}$ is the endpoint of at most one segment and (2) two no segments intersect. (The configuration with no edges drawn is allowed.)",
        "answer": "323",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1633,
        "difficulty": 5.0
      },
      {
        "id": "af4eaa3c5c3bd506a891219a82ea1f259043d2c0dbfbedb80749802b861e4e05",
        "domain": "omni",
        "question": "For a string of $P$ 's and $Q$ 's, the value is defined to be the product of the positions of the $P$ 's. For example, the string $P P Q P Q Q$ has value $1 \\cdot 2 \\cdot 4=8$. Also, a string is called antipalindromic if writing it backwards, then turning all the $P$ 's into $Q$ 's and vice versa, produces the original string. For example, $P P Q P Q Q$ is antipalindromic. There are $2^{1002}$ antipalindromic strings of length 2004. Find the sum of the reciprocals of their values.",
        "answer": "2005^{1002} / 2004!",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 824,
        "difficulty": 5.25
      },
      {
        "id": "ed0b66e8e6654018d148a4ef4f9b0e647273bfbd1dc5e39c9abf89f9d1f16acd",
        "domain": "omni",
        "question": "Suppose that a positive integer $N$ can be expressed as the sum of $k$ consecutive positive integers \\[ N = a + (a+1) +(a+2) + \\cdots + (a+k-1) \\] for $k=2017$ but for no other values of $k>1$. Considering all positive integers $N$ with this property, what is the smallest positive integer $a$ that occurs in any of these expressions?",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Factorization",
        "source": "putnam",
        "source_index": 3250,
        "difficulty": 7.0
      },
      {
        "id": "ba7ac9688773dfb6f6a230a7e176bb0303a7487bd0036436816b63ac33795bae",
        "domain": "omni",
        "question": "Calculate the value of $(3,1) \\nabla (4,2)$ using the operation ' $\\nabla$ ' defined by $(a, b) \\nabla (c, d)=ac+bd$.",
        "answer": "14",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3149,
        "difficulty": 1.0
      },
      {
        "id": "3cc95e8fddb311c66c8325a781d233fb035334d596c107a0419da65a45dd115b",
        "domain": "omni",
        "question": "Consider a square, inside which is inscribed a circle, inside which is inscribed a square, inside which is inscribed a circle, and so on, with the outermost square having side length 1. Find the difference between the sum of the areas of the squares and the sum of the areas of the circles.",
        "answer": "2 - \\frac{\\pi}{2}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area; Mathematics -> Calculus -> Integral Calculus -> Series -> Other",
        "source": "HMMT_11",
        "source_index": 1829,
        "difficulty": 4.5
      },
      {
        "id": "da7ff2156b2f10a5ffd88aa341c254dca482310c6acfc25b6c37542ee8e94c5d",
        "domain": "omni",
        "question": "How many students chose Greek food if 200 students were asked to choose between pizza, Thai food, or Greek food, and the circle graph shows the results?",
        "answer": "100",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "pascal",
        "source_index": 2995,
        "difficulty": 1.0
      },
      {
        "id": "23b52d0cee75a44472076cf658eb273c7740a0b44e467f87798136b79e2e2872",
        "domain": "omni",
        "question": "For each integer $x$ with $1 \\leq x \\leq 10$, a point is randomly placed at either $(x, 1)$ or $(x,-1)$ with equal probability. What is the expected area of the convex hull of these points? Note: the convex hull of a finite set is the smallest convex polygon containing it.",
        "answer": "\\frac{1793}{128}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 1236,
        "difficulty": 4.5
      },
      {
        "id": "4ce4fc5a81dcc05f4c86248d63088d5934c683c10096090d93fd94c44aef48a9",
        "domain": "omni",
        "question": "Turbo the snail sits on a point on a circle with circumference $1$. Given an infinite sequence of positive real numbers $c_1, c_2, c_3, \\dots$, Turbo successively crawls distances $c_1, c_2, c_3, \\dots$ around the circle, each time choosing to crawl either clockwise or counterclockwise.\nDetermine the largest constant $C > 0$ with the following property: for every sequence of positive real numbers $c_1, c_2, c_3, \\dots$ with $c_i < C$ for all $i$, Turbo can (after studying the sequence) ensure that there is some point on the circle that it will never visit or crawl across.",
        "answer": "0.5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "european_girls_mo",
        "source_index": 3628,
        "difficulty": 8.0
      },
      {
        "id": "0be490b8782ab0f6b954ecd3f8a786f0f6355cc693749d972863c41077d233cb",
        "domain": "omni",
        "question": "Let $A B C D$ be a convex quadrilateral inscribed in a circle with shortest side $A B$. The ratio $[B C D] /[A B D]$ is an integer (where $[X Y Z]$ denotes the area of triangle $X Y Z$.) If the lengths of $A B, B C, C D$, and $D A$ are distinct integers no greater than 10, find the largest possible value of $A B$.",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 986,
        "difficulty": 4.0
      },
      {
        "id": "e8fe79ec6fa03415c511514d52b2ea42cae8a12506bec7b85c3f74d9525a3c17",
        "domain": "omni",
        "question": "Let \\(\\triangle A B C\\) be a right triangle with right angle \\(C\\). Let \\(I\\) be the incenter of \\(A B C\\), and let \\(M\\) lie on \\(A C\\) and \\(N\\) on \\(B C\\), respectively, such that \\(M, I, N\\) are collinear and \\(\\overline{M N}\\) is parallel to \\(A B\\). If \\(A B=36\\) and the perimeter of \\(C M N\\) is 48, find the area of \\(A B C\\).",
        "answer": "252",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2129,
        "difficulty": 6.0
      },
      {
        "id": "588d807ba0c0a94ffbb523fd9f5c3bce4c22c317ee53befc0fa65c3e5b5c4d4f",
        "domain": "omni",
        "question": "A function $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ satisfies: $f(0)=0$ and $$\\left|f\\left((n+1) 2^{k}\\right)-f\\left(n 2^{k}\\right)\\right| \\leq 1$$ for all integers $k \\geq 0$ and $n$. What is the maximum possible value of $f(2019)$?",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1882,
        "difficulty": 4.5
      },
      {
        "id": "0b0033ad2652c93b5908f3118409fd7f523216749fa24740cb3c07416d4c3f74",
        "domain": "omni",
        "question": "Karen has seven envelopes and seven letters of congratulations to various HMMT coaches. If she places the letters in the envelopes at random with each possible configuration having an equal probability, what is the probability that exactly six of the letters are in the correct envelopes?",
        "answer": "0",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 447,
        "difficulty": 3.0
      },
      {
        "id": "10c909e88870fe7e451fa9d4afc2284fc874b2640bdacfebf0dd43a9336c7abd",
        "domain": "omni",
        "question": "$S$ is a set of complex numbers such that if $u, v \\in S$, then $u v \\in S$ and $u^{2}+v^{2} \\in S$. Suppose that the number $N$ of elements of $S$ with absolute value at most 1 is finite. What is the largest possible value of $N$ ?",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 724,
        "difficulty": 5.0
      },
      {
        "id": "67abeefc0c91f3bf0faec5a0568582513fb3d26a728c41b93bebbd43cb93a481",
        "domain": "omni",
        "question": "Two mathematicians, Kelly and Jason, play a cooperative game. The computer selects some secret positive integer $n<60$ (both Kelly and Jason know that $n<60$, but that they don't know what the value of $n$ is). The computer tells Kelly the unit digit of $n$, and it tells Jason the number of divisors of $n$. Then, Kelly and Jason have the following dialogue: Kelly: I don't know what $n$ is, and I'm sure that you don't know either. However, I know that $n$ is divisible by at least two different primes. Jason: Oh, then I know what the value of $n$ is. Kelly: Now I also know what $n$ is. Assuming that both Kelly and Jason speak truthfully and to the best of their knowledge, what are all the possible values of $n$?",
        "answer": "10",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 497,
        "difficulty": 4.5
      },
      {
        "id": "74b6346a74941389372502b7e748b1c93b99c63b4dc122d068d293289c7af16d",
        "domain": "omni",
        "question": "An integer $n$ is chosen uniformly at random from the set $\\{1,2,3, \\ldots, 2023!\\}$. Compute the probability that $$\\operatorname{gcd}\\left(n^{n}+50, n+1\\right)=1$$",
        "answer": "\\frac{265}{357}",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 1823,
        "difficulty": 4.5
      },
      {
        "id": "11eef74f234071ab8085a990d71325eec276ca084d85f6347f6056c21e20f89a",
        "domain": "omni",
        "question": "Ewan writes out a sequence where he counts by 11s starting at 3. Which number will appear in Ewan's sequence?",
        "answer": "113",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "fermat",
        "source_index": 3489,
        "difficulty": 1.5
      },
      {
        "id": "510b04af63db14d14352cb6e8197d9df86ea08c0fdaa0537dab46cd4d412de0d",
        "domain": "omni",
        "question": "A number $n$ is $b a d$ if there exists some integer $c$ for which $x^{x} \\equiv c(\\bmod n)$ has no integer solutions for $x$. Find the number of bad integers between 2 and 42 inclusive.",
        "answer": "25",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2484,
        "difficulty": 4.0
      },
      {
        "id": "66358b40c08150a3de3ae1df23010a1a14c4582d44eb5c4c81b4667fdea494e2",
        "domain": "omni",
        "question": "In unit square $A B C D$, points $E, F, G$ are chosen on side $B C, C D, D A$ respectively such that $A E$ is perpendicular to $E F$ and $E F$ is perpendicular to $F G$. Given that $G A=\\frac{404}{1331}$, find all possible values of the length of $B E$.",
        "answer": "\\frac{9}{11}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2580,
        "difficulty": 4.0
      },
      {
        "id": "efeea95bc2d923ef697f2fca679c941cd95814f596eb655f9ed658fc5300393d",
        "domain": "omni",
        "question": "As part of his effort to take over the world, Edward starts producing his own currency. As part of an effort to stop Edward, Alex works in the mint and produces 1 counterfeit coin for every 99 real ones. Alex isn't very good at this, so none of the counterfeit coins are the right weight. Since the mint is not perfect, each coin is weighed before leaving. If the coin is not the right weight, then it is sent to a lab for testing. The scale is accurate $95 \\%$ of the time, $5 \\%$ of all the coins minted are sent to the lab, and the lab's test is accurate $90 \\%$ of the time. If the lab says a coin is counterfeit, what is the probability that it really is?",
        "answer": "\\frac{19}{28}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1219,
        "difficulty": 4.0
      },
      {
        "id": "035bfa604133d1f9efefc439aefc82b23564f6fb3810f5dfd888f7067b2974a3",
        "domain": "omni",
        "question": "Define the polynomials $P_0, P_1, P_2 \\cdots$ by:\n\\[ P_0(x)=x^3+213x^2-67x-2000 \\]\n\\[ P_n(x)=P_{n-1}(x-n), n \\in N \\]\nFind the coefficient of $x$ in $P_{21}(x)$.",
        "answer": "61610",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "pan_african MO",
        "source_index": 3790,
        "difficulty": 6.0
      },
      {
        "id": "64da8df4cf5e449039706ab844ac2a34aa5bc0ec8e348752c8bd76789b44070b",
        "domain": "omni",
        "question": "There are 42 stepping stones in a pond, arranged along a circle. You are standing on one of the stones. You would like to jump among the stones so that you move counterclockwise by either 1 stone or 7 stones at each jump. Moreover, you would like to do this in such a way that you visit each stone (except for the starting spot) exactly once before returning to your initial stone for the first time. In how many ways can you do this?",
        "answer": "63",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1045,
        "difficulty": 5.25
      },
      {
        "id": "0daf9c54ffd148ca1e990692bc5b89998af5210b6b7de45de22d373255b797be",
        "domain": "omni",
        "question": "Determine the remainder when $$\\sum_{i=0}^{2015}\\left\\lfloor\\frac{2^{i}}{25}\\right\\rfloor$$ is divided by 100, where $\\lfloor x\\rfloor$ denotes the largest integer not greater than $x$.",
        "answer": "14",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1356,
        "difficulty": 4.5
      },
      {
        "id": "f82f973ac22b463fe2a6ee2b9b1ac236d73fa9b611a6a1ef7f685699465b6e7f",
        "domain": "omni",
        "question": "Determine the sum of all distinct real values of $x$ such that $|||\\cdots||x|+x|\\cdots|+x|+x|=1$ where there are 2017 $x$ 's in the equation.",
        "answer": "-\\frac{2016}{2017}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2131,
        "difficulty": 4.0
      },
      {
        "id": "d3a18005ab297a48fbc58b1ba1b9d3a99818fb5e384e13f66ae265bf4702f442",
        "domain": "omni",
        "question": "Let $n$ be a positive integer, and let $s$ be the sum of the digits of the base-four representation of $2^{n}-1$. If $s=2023$ (in base ten), compute $n$ (in base ten).",
        "answer": "1349",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Abstract Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1539,
        "difficulty": 5.5
      },
      {
        "id": "aed9fbdb24e39fdb85aa6ee1fa5fd452d013166e49a393474208659c6fb92d8c",
        "domain": "omni",
        "question": "Let $A B C D E$ be a convex pentagon such that $$\\begin{aligned} & A B+B C+C D+D E+E A=64 \\text { and } \\\\ & A C+C E+E B+B D+D A=72 \\end{aligned}$$ Compute the perimeter of the convex pentagon whose vertices are the midpoints of the sides of $A B C D E$.",
        "answer": "36",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1756,
        "difficulty": 3.5
      },
      {
        "id": "5ecf4f59e3ccf7883dfcc5400a31fae996973912bdbfda75ffb7923cce712045",
        "domain": "omni",
        "question": "For how many pairs $(m, n)$ with $m$ and $n$ integers satisfying $1 \\leq m \\leq 100$ and $101 \\leq n \\leq 205$ is $3^{m}+7^{n}$ divisible by 10?",
        "answer": "2625",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "cayley",
        "source_index": 3060,
        "difficulty": 2.5
      },
      {
        "id": "b75dfea3848010c37488caa88d72b50b9986c625715c9e75337bbb00b0009571",
        "domain": "omni",
        "question": "In convex quadrilateral \\(ABCD\\) with \\(AB=11\\) and \\(CD=13\\), there is a point \\(P\\) for which \\(\\triangle ADP\\) and \\(\\triangle BCP\\) are congruent equilateral triangles. Compute the side length of these triangles.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2337,
        "difficulty": 5.0
      },
      {
        "id": "6128c25ccf677594fa84e5cd47b246b589589b3ea87b229e5d4ca8e59eb4d245",
        "domain": "omni",
        "question": "Alex has a $20 \\times 16$ grid of lightbulbs, initially all off. He has 36 switches, one for each row and column. Flipping the switch for the $i$th row will toggle the state of each lightbulb in the $i$th row (so that if it were on before, it would be off, and vice versa). Similarly, the switch for the $j$th column will toggle the state of each bulb in the $j$th column. Alex makes some (possibly empty) sequence of switch flips, resulting in some configuration of the lightbulbs and their states. How many distinct possible configurations of lightbulbs can Alex achieve with such a sequence? Two configurations are distinct if there exists a lightbulb that is on in one configuration and off in another.",
        "answer": "2^{35}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2011,
        "difficulty": 4.5
      },
      {
        "id": "5b7257fadb754705de0ce14b59b29e0acb7c161872a719eacdeb1ec8595c6f1b",
        "domain": "omni",
        "question": "For what value of $k$ is the line through the points $(3,2k+1)$ and $(8,4k-5)$ parallel to the $x$-axis?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3395,
        "difficulty": 1.0
      },
      {
        "id": "1a97440baf5f920817d9e44acb20ea40715e4fafc52e32cb01c112f6f9e0d12b",
        "domain": "omni",
        "question": "Let $f(n)$ be the number of ways to write $n$ as a sum of powers of $2$ , where we keep track of the order of the summation.  For example, $f(4)=6$ because $4$ can be written as $4$ , $2+2$ , $2+1+1$ , $1+2+1$ , $1+1+2$ , and $1+1+1+1$ .  Find the smallest $n$ greater than $2013$ for which $f(n)$ is odd.",
        "answer": "\\[ 2047 \\]",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Number Theory -> Prime Numbers",
        "source": "usajmo",
        "source_index": 195,
        "difficulty": 5.5
      },
      {
        "id": "6d90f607aa900f871d61124205600b62206c577d6fa4ea905c4978619497b2c5",
        "domain": "omni",
        "question": "Let $z$ be a complex number and $k$ a positive integer such that $z^{k}$ is a positive real number other than 1. Let $f(n)$ denote the real part of the complex number $z^{n}$. Assume the parabola $p(n)=an^{2}+bn+c$ intersects $f(n)$ four times, at $n=0,1,2,3$. Assuming the smallest possible value of $k$, find the largest possible value of $a$.",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers; Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 2204,
        "difficulty": 4.5
      },
      {
        "id": "2887abd8148838bc053f288d2cc45fb8517c81a55a2e84c0ef7ecb29f2d95746",
        "domain": "omni",
        "question": "Let $S$ be a set of positive integers satisfying the following two conditions: - For each positive integer $n$, at least one of $n, 2 n, \\ldots, 100 n$ is in $S$. - If $a_{1}, a_{2}, b_{1}, b_{2}$ are positive integers such that $\\operatorname{gcd}\\left(a_{1} a_{2}, b_{1} b_{2}\\right)=1$ and $a_{1} b_{1}, a_{2} b_{2} \\in S$, then $a_{2} b_{1}, a_{1} b_{2} \\in S$ Suppose that $S$ has natural density $r$. Compute the minimum possible value of $\\left\\lfloor 10^{5} r\\right\\rfloor$. Note: $S$ has natural density $r$ if $\\frac{1}{n}|S \\cap\\{1, \\ldots, n\\}|$ approaches $r$ as $n$ approaches $\\infty$.",
        "answer": "396",
        "topic": "Mathematics -> Number Theory -> Congruences; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1474,
        "difficulty": 5.0
      },
      {
        "id": "ff7b4edd4a43d21fe808ea619300d18264f4e0bbaaa58217b7f8a16d5bd412b0",
        "domain": "omni",
        "question": "What is the value of \\( \\frac{5-2}{2+1} \\)?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 2823,
        "difficulty": 1.0
      },
      {
        "id": "f90664e84faf06bf79f859113e8b2d3a2a60b7a1d1534410bb28b2465f4c81ee",
        "domain": "omni",
        "question": "Determine which of the following expressions has the largest value: $4^2$, $4 \\times 2$, $4 - 2$, $\\frac{4}{2}$, or $4 + 2$.",
        "answer": "16",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2898,
        "difficulty": 1.0
      },
      {
        "id": "c00c079a2cde89679d48e465a064672def94ffc250940cd66b336225a96880ea",
        "domain": "omni",
        "question": "For each real number $x$, let\n\\[\nf(x) = \\sum_{n\\in S_x} \\frac{1}{2^n},\n\\]\nwhere $S_x$ is the set of positive integers $n$ for which $\\lfloor nx \\rfloor$ is even. What is the largest real number $L$ such that $f(x) \\geq L$ for all $x \\in [0,1)$? (As usual, $\\lfloor z \\rfloor$ denotes the greatest integer less than or equal to $z$.)",
        "answer": "4/7",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Precalculus -> Functions; Mathematics -> Discrete Mathematics -> Algorithms",
        "source": "putnam",
        "source_index": 3207,
        "difficulty": 7.0
      },
      {
        "id": "a54ed15b3d68b22cd70f622bb1fed976096a9d5d7d5a9588e667f4cf7bcd4661",
        "domain": "omni",
        "question": "For positive integers $a$ and $b$ such that $a$ is coprime to $b$, define $\\operatorname{ord}_{b}(a)$ as the least positive integer $k$ such that $b \\mid a^{k}-1$, and define $\\varphi(a)$ to be the number of positive integers less than or equal to $a$ which are coprime to $a$. Find the least positive integer $n$ such that $$\\operatorname{ord}_{n}(m)<\\frac{\\varphi(n)}{10}$$ for all positive integers $m$ coprime to $n$.",
        "answer": "240",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 1016,
        "difficulty": 5.0
      },
      {
        "id": "594072fbe40032a15389b515e815090828e932b85f2254851f304e27b6bd1c6b",
        "domain": "omni",
        "question": "Count the number of functions $f: \\mathbb{Z} \\rightarrow\\{$ 'green', 'blue' $\\}$ such that $f(x)=f(x+22)$ for all integers $x$ and there does not exist an integer $y$ with $f(y)=f(y+2)=$ 'green'.",
        "answer": "39601",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 3342,
        "difficulty": 5.0
      },
      {
        "id": "3c44d44e18e9c64edf75a9518b597f71c02f17cfef7c09b9653af67bc9eb8225",
        "domain": "omni",
        "question": "Given a positive integer $n$, let $M(n)$ be the largest integer $m$ such that \\[ \\binom{m}{n-1} > \\binom{m-1}{n}. \\] Evaluate \\[ \\lim_{n \\to \\infty} \\frac{M(n)}{n}. \\]",
        "answer": "\\frac{3+\\sqrt{5}}{2}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Precalculus -> Limits",
        "source": "putnam",
        "source_index": 3184,
        "difficulty": 8.0
      },
      {
        "id": "745fd9c84fcd872d52630c58f833bb67c92aaec6730e66e0c8a0a049172482f4",
        "domain": "omni",
        "question": "FemtoPravis is walking on an $8 \\times 8$ chessboard that wraps around at its edges (so squares on the left edge of the chessboard are adjacent to squares on the right edge, and similarly for the top and bottom edges). Each femtosecond, FemtoPravis moves in one of the four diagonal directions uniformly at random. After 2012 femtoseconds, what is the probability that FemtoPravis is at his original location?",
        "answer": "\\left(\\frac{1+2^{1005}}{2^{1007}}\\right)^{2}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 805,
        "difficulty": 5.0
      },
      {
        "id": "8780cc0a30c27f96d6f8f72ad6d95a2099d4b7af47abafe65c0ade951c67a9b1",
        "domain": "omni",
        "question": "Compute the smallest positive integer $n$ for which $$0<\\sqrt[4]{n}-\\lfloor\\sqrt[4]{n}\\rfloor<\\frac{1}{2015}$$",
        "answer": "4097",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_11",
        "source_index": 1955,
        "difficulty": 3.5
      },
      {
        "id": "a8289d61ee41c0e05f1d1b698608dd56a919b2312ea3b862d244409e5e9bd657",
        "domain": "omni",
        "question": "Write 1 as a sum of 4 distinct unit fractions.",
        "answer": "\\frac{1}{2}+\\frac{1}{3}+\\frac{1}{7}+\\frac{1}{42}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_11",
        "source_index": 3373,
        "difficulty": 4.5
      },
      {
        "id": "b27b70bb8aa98e1ba74ff7448750e52045d0c78d13ce411d1c63893d2a5b73ba",
        "domain": "omni",
        "question": "How many regions of the plane are bounded by the graph of $$x^{6}-x^{5}+3 x^{4} y^{2}+10 x^{3} y^{2}+3 x^{2} y^{4}-5 x y^{4}+y^{6}=0 ?$$",
        "answer": "5",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Geometry -> Differential Geometry -> Curvature",
        "source": "HMMT_2",
        "source_index": 1000,
        "difficulty": 5.0
      },
      {
        "id": "26ef7d2abe60e21ba322356e7844c2feeb8f9381f318fc3fd0c7cee0e67e0187",
        "domain": "omni",
        "question": "Let $N$ be the largest positive integer that can be expressed as a 2013-digit base -4 number. What is the remainder when $N$ is divided by 210?",
        "answer": "51",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2267,
        "difficulty": 4.0
      },
      {
        "id": "b0f3cbaad5e14675fa9363f3acb545d298368994b249ca113d38077711cdd17b",
        "domain": "omni",
        "question": "What is the integer formed by the rightmost two digits of the integer equal to \\(4^{127} + 5^{129} + 7^{131}\\)?",
        "answer": "52",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "pascal",
        "source_index": 3079,
        "difficulty": 2.5
      },
      {
        "id": "477433740877d92458f69ceb05b79cf4ffb36e55f97e457ab36c4690297ad56b",
        "domain": "omni",
        "question": "Kristoff is planning to transport a number of indivisible ice blocks with positive integer weights from the north mountain to Arendelle. He knows that when he reaches Arendelle, Princess Anna and Queen Elsa will name an ordered pair $(p, q)$ of nonnegative integers satisfying $p+q \\leq 2016$. Kristoff must then give Princess Anna exactly $p$ kilograms of ice. Afterward, he must give Queen Elsa exactly $q$ kilograms of ice. What is the minimum number of blocks of ice Kristoff must carry to guarantee that he can always meet Anna and Elsa's demands, regardless of which $p$ and $q$ are chosen?",
        "answer": "18",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 1160,
        "difficulty": 5.5
      },
      {
        "id": "59767ca4fec14bc134375a899b1b37152e663ff006beb02d403936a9b7d0fdbf",
        "domain": "omni",
        "question": "There are $2n$ students in a school $(n \\in \\mathbb{N}, n \\geq 2)$. Each week $n$ students go on a trip. After several trips the following condition was fulfilled: every two students were together on at least one trip. What is the minimum number of trips needed for this to happen?",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "imc",
        "source_index": 2541,
        "difficulty": 6.0
      },
      {
        "id": "c196b6c7c6955678ac7cc5673c4cbb994c80a154b1bc82375cb64d12e94dd403",
        "domain": "omni",
        "question": "Find the largest real number $\\lambda$ such that $a^{2}+b^{2}+c^{2}+d^{2} \\geq a b+\\lambda b c+c d$ for all real numbers $a, b, c, d$.",
        "answer": "\\frac{3}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "HMMT_11",
        "source_index": 2189,
        "difficulty": 4.0
      },
      {
        "id": "e91d742b0f763e18c9343c559460717cd4bfc29b183e0518f4d25cf78ddc3481",
        "domain": "omni",
        "question": "What is the perimeter of the figure shown if $x=3$?",
        "answer": "23",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "pascal",
        "source_index": 2838,
        "difficulty": 1.0
      },
      {
        "id": "0d5d8d816e4e1af1f25ad3a18bf919f1615a6e9204d1763702203f859fd60634",
        "domain": "omni",
        "question": "Let $\\Gamma$ denote the circumcircle of triangle $A B C$. Point $D$ is on $\\overline{A B}$ such that $\\overline{C D}$ bisects $\\angle A C B$. Points $P$ and $Q$ are on $\\Gamma$ such that $\\overline{P Q}$ passes through $D$ and is perpendicular to $\\overline{C D}$. Compute $P Q$, given that $B C=20, C A=80, A B=65$.",
        "answer": "4 \\sqrt{745}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 662,
        "difficulty": 5.0
      },
      {
        "id": "7ccfa6f022e29d0d2e8abc6df95430248ab3d9b747614c5904d4a0ae141c89b8",
        "domain": "omni",
        "question": "Find $PB$ given that $AP$ is a tangent to $\\Omega$, $\\angle PAB=\\angle PCA$, and $\\frac{PB}{PA}=\\frac{4}{7}=\\frac{PA}{PB+6}$.",
        "answer": "\\frac{32}{11}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Equations and Inequalities -> Other",
        "source": "HMMT_11",
        "source_index": 2375,
        "difficulty": 5.0
      },
      {
        "id": "d3c54489cbd2aea157e5a27010eb1074ce0c889f4523a52e71891902da3063d4",
        "domain": "omni",
        "question": "The altitudes of a triangle are $12$, $15$, and $20$.  What is the area of this triangle?",
        "answer": "150",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "baltic_way",
        "source_index": 3623,
        "difficulty": 5.0
      },
      {
        "id": "05fb3a3eb0320c587dd87d3a50dd151b9388382c68efc78187cddead40ddb2fc",
        "domain": "omni",
        "question": "A right triangle has side lengths $a, b$, and $\\sqrt{2016}$ in some order, where $a$ and $b$ are positive integers. Determine the smallest possible perimeter of the triangle.",
        "answer": "48+\\sqrt{2016}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 992,
        "difficulty": 5.0
      },
      {
        "id": "2c6345849f533d7231a644fa0296f8045712598a53b4784d2755c0d0bdd0243d",
        "domain": "omni",
        "question": "In this final problem, a ball is again launched from the vertex of an equilateral triangle with side length 5. In how many ways can the ball be launched so that it will return again to a vertex for the first time after 2009 bounces?",
        "answer": "502",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1778,
        "difficulty": 5.0
      },
      {
        "id": "e61d96855fcc6ad005e9d0da14ab978d4fec99e7864b14a8a7d3fcd7cffcf056",
        "domain": "omni",
        "question": "Sergio recently opened a store. One day, he determined that the average number of items sold per employee to date was 75. The next day, one employee sold 6 items, one employee sold 5 items, and one employee sold 4 items. The remaining employees each sold 3 items. This made the new average number of items sold per employee to date equal to 78.3. How many employees are there at the store?",
        "answer": "20",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2809,
        "difficulty": 2.0
      },
      {
        "id": "40e6f703f921e4d747c1af0fde8f44cc7973996a049e6e898c730281efed0596",
        "domain": "omni",
        "question": "A square grid $100 \\times 100$ is tiled in two ways - only with dominoes and only with squares $2 \\times 2$. What is the least number of dominoes that are entirely inside some square $2 \\times 2$?",
        "answer": "100",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "problems_from_the_kvant_magazine",
        "source_index": 3733,
        "difficulty": 4.5
      },
      {
        "id": "94faa58f4c965f3a646dad61ab1d4f245df210244249ca6736bf7ca1632c475b",
        "domain": "omni",
        "question": "Each of the four digits of the integer 2024 is even. How many integers between 1000 and 9999, inclusive, have the property that all four of their digits are even?",
        "answer": "500",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 2702,
        "difficulty": 2.0
      },
      {
        "id": "627ecdfcffbf2b29f0a12bc33666a25373a3ceb9f708fbcda4bf47d8d16b3745",
        "domain": "omni",
        "question": "A string has been cut into 4 pieces, all of different lengths. The length of each piece is 2 times the length of the next smaller piece. What fraction of the original string is the longest piece?",
        "answer": "\\frac{8}{15}",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3436,
        "difficulty": 1.5
      },
      {
        "id": "b47c79eccee3a3c666bcd221c1a8c59502c265f2f3e48fa2461c69853f4c5fce",
        "domain": "omni",
        "question": "A positive integer \\overline{A B C}, where $A, B, C$ are digits, satisfies $\\overline{A B C}=B^{C}-A$. Find $\\overline{A B C}$.",
        "answer": "127",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2056,
        "difficulty": 4.0
      },
      {
        "id": "bc6710413d35470e4cab25dc1409af6501c1a0af21760b6504ea5b0e889033d7",
        "domain": "omni",
        "question": "A sequence of positive integers is given by $a_{1}=1$ and $a_{n}=\\operatorname{gcd}\\left(a_{n-1}, n\\right)+1$ for $n>1$. Calculate $a_{2002}$.",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 940,
        "difficulty": 4.5
      },
      {
        "id": "cbc80fe8153d37085bf329dc55261bd59224680912d57c3e10072d5613374915",
        "domain": "omni",
        "question": "A hotel consists of a $2 \\times 8$ square grid of rooms, each occupied by one guest. All the guests are uncomfortable, so each guest would like to move to one of the adjoining rooms (horizontally or vertically). Of course, they should do this simultaneously, in such a way that each room will again have one guest. In how many different ways can they collectively move?",
        "answer": "1156",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 457,
        "difficulty": 5.25
      },
      {
        "id": "09417909a3d00b650b48a07f431ea6f0ad8a2fb3ee4bb7239d105e3bd19860a7",
        "domain": "omni",
        "question": "Jim wrote a sequence of symbols a total of 50 times. How many more of one symbol than another did he write?",
        "answer": "150",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2902,
        "difficulty": 1.5
      },
      {
        "id": "db0fef4083ab658d2e8266d3f3bdc308c14417c0125794a525af1e483111edd0",
        "domain": "omni",
        "question": "The L shape made by adjoining three congruent squares can be subdivided into four smaller L shapes. Each of these can in turn be subdivided, and so forth. If we perform 2005 successive subdivisions, how many of the $4^{2005}$ L's left at the end will be in the same orientation as the original one?",
        "answer": "4^{2004}+2^{2004}",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 973,
        "difficulty": 4.25
      },
      {
        "id": "e3b837c4a7b97d97fb0f50a3cc043f15299cfbb7b96476a098f594724bec4f1e",
        "domain": "omni",
        "question": "Find the number of integers $n$ with $1 \\leq n \\leq 2017$ so that $(n-2)(n-0)(n-1)(n-7)$ is an integer multiple of 1001.",
        "answer": "99",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2147,
        "difficulty": 4.0
      },
      {
        "id": "408af27b9ba5851d1c4b62847f7c42e127437b23ac6037b514129d0f555a530e",
        "domain": "omni",
        "question": "In acute triangle $ABC$, let $H$ be the orthocenter and $D$ the foot of the altitude from $A$. The circumcircle of triangle $BHC$ intersects $AC$ at $E \\neq C$, and $AB$ at $F \\neq B$. If $BD=3, CD=7$, and $\\frac{AH}{HD}=\\frac{5}{7}$, the area of triangle $AEF$ can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$.",
        "answer": "12017",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2548,
        "difficulty": 5.25
      },
      {
        "id": "ef9f09e7a68d26dbed82a83991e5d09e73fbd91757faae143204881f2a941f8d",
        "domain": "omni",
        "question": "For each continuous function $f: [0,1] \\to \\mathbb{R}$, let $I(f) = \\int_0^1 x^2 f(x)\\,dx$ and $J(x) = \\int_0^1 x \\left(f(x)\\right)^2\\,dx$. Find the maximum value of $I(f) - J(f)$ over all such functions $f$.",
        "answer": "1/16",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals",
        "source": "putnam",
        "source_index": 3258,
        "difficulty": 7.0
      },
      {
        "id": "fe549fa4189e2117a9151c5bf98b2b2c47c1fea73b8ebac8148a3a6d72703a2c",
        "domain": "omni",
        "question": "A subset $S$ of the nonnegative integers is called supported if it contains 0, and $k+8, k+9 \\in S$ for all $k \\in S$. How many supported sets are there?",
        "answer": "1430",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1368,
        "difficulty": 6.5
      },
      {
        "id": "006caa7786b8d2332f20884da997862e9c91a8aba8bc5a2f3fa21f4cdc42e85c",
        "domain": "omni",
        "question": "Let $x$ be a positive real number. Find the maximum possible value of $$\\frac{x^{2}+2-\\sqrt{x^{4}+4}}{x}$$",
        "answer": "2 \\sqrt{2}-2",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "HMMT_2",
        "source_index": 1358,
        "difficulty": 5.0
      },
      {
        "id": "e41ad274e26fd78c9f9fa85b36e8a66028bccb6d0111a927d62fe7548ca15683",
        "domain": "omni",
        "question": "The area of the largest regular hexagon that can fit inside of a rectangle with side lengths 20 and 22 can be expressed as $a \\sqrt{b}-c$, for positive integers $a, b$, and $c$, where $b$ is squarefree. Compute $100 a+10 b+c$.",
        "answer": "134610",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 1324,
        "difficulty": 5.0
      },
      {
        "id": "47b5d9598a28f7d7a648ccee612c2f8bf2455d0f35372ab2538ab171d97bfdf9",
        "domain": "omni",
        "question": "For integers $a, b, c, d$, let $f(a, b, c, d)$ denote the number of ordered pairs of integers $(x, y) \\in \\{1,2,3,4,5\\}^{2}$ such that $a x+b y$ and $c x+d y$ are both divisible by 5. Find the sum of all possible values of $f(a, b, c, d)$.",
        "answer": "31",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Field Theory; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 847,
        "difficulty": 4.0
      },
      {
        "id": "201aac1af15cb946d555e493fa3ad68cb2c547ccf5cd054b5e4d0a05d9000937",
        "domain": "omni",
        "question": "Crisp All, a basketball player, is dropping dimes and nickels on a number line. Crisp drops a dime on every positive multiple of 10 , and a nickel on every multiple of 5 that is not a multiple of 10. Crisp then starts at 0 . Every second, he has a $\\frac{2}{3}$ chance of jumping from his current location $x$ to $x+3$, and a $\\frac{1}{3}$ chance of jumping from his current location $x$ to $x+7$. When Crisp jumps on either a dime or a nickel, he stops jumping. What is the probability that Crisp stops on a dime?",
        "answer": "\\frac{20}{31}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2363,
        "difficulty": 4.0
      },
      {
        "id": "44fc831985e1602793ce73f56a27f1d64683fae962b4612d7a36a28b397fe50e",
        "domain": "omni",
        "question": "In a party with $1982$ people, among any group of four there is at least one person who knows each of the other three. What is the minimum number of people in the party who know everyone else?",
        "answer": "\\[ 1979 \\]",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "usamo",
        "source_index": 327,
        "difficulty": 7.0
      },
      {
        "id": "d29275d7e27a2560c443ff164b7a00086158924218086de58f5d874987ff6708",
        "domain": "omni",
        "question": "Let $ ABP, BCQ, CAR$ be three non-overlapping triangles erected outside of acute triangle $ ABC$. Let $ M$ be the midpoint of segment $ AP$. Given that $ \\angle PAB \\equal{} \\angle CQB \\equal{} 45^\\circ$, $ \\angle ABP \\equal{} \\angle QBC \\equal{} 75^\\circ$, $ \\angle RAC \\equal{} 105^\\circ$, and $ RQ^2 \\equal{} 6CM^2$, compute $ AC^2/AR^2$.\r\n\r\n[i]Zuming Feng.[/i]",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangles -> Other; Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "usa_team_selection_test",
        "source_index": 17,
        "difficulty": 6.5
      },
      {
        "id": "137dfb78e8c47a4eecfe1564378cb1b8356539ba1234d6f4822cb8b41c8558a1",
        "domain": "omni",
        "question": "For non-negative real numbers $x_1, x_2, \\ldots, x_n$ which satisfy $x_1 + x_2 + \\cdots + x_n = 1$, find the largest possible value of $\\sum_{j = 1}^{n} (x_j^{4} - x_j^{5})$.",
        "answer": "\\frac{1}{12}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "china_team_selection_test",
        "source_index": 441,
        "difficulty": 7.5
      },
      {
        "id": "199756ce26852fbf1980e14ae4a280b20bebfd6bd176a4876a2122fd93aa2ebc",
        "domain": "omni",
        "question": "In the sum shown, each letter represents a different digit with $T \\neq 0$ and $W \\neq 0$. How many different values of $U$ are possible? \n\n\\begin{tabular}{rrrrr} \n& $W$ & $X$ & $Y$ & $Z$ \\\\ \n+ & $W$ & $X$ & $Y$ & $Z$ \\\\ \n\\hline & $W$ & $U$ & $Y$ & $V$ \n\\end{tabular}",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2820,
        "difficulty": 2.5
      },
      {
        "id": "595b63a89df7a08a7cdca2e9e97b975039b1719680036ce366bdafc3931b641a",
        "domain": "omni",
        "question": "Rosencrantz and Guildenstern are playing a game where they repeatedly flip coins. Rosencrantz wins if 1 heads followed by 2009 tails appears. Guildenstern wins if 2010 heads come in a row. They will flip coins until someone wins. What is the probability that Rosencrantz wins?",
        "answer": "$\\sqrt{\\frac{2^{2009}-1}{3 \\cdot 2^{2008}-1}}$",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1244,
        "difficulty": 7.0
      },
      {
        "id": "591a9beb65317d68b31b036c8afa8021fe29ba4b6e8c0f164becbaa0d49418b1",
        "domain": "omni",
        "question": "Determine the number of integers $D$ such that whenever $a$ and $b$ are both real numbers with $-1 / 4<a, b<1 / 4$, then $\\left|a^{2}-D b^{2}\\right|<1$.",
        "answer": "32",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1954,
        "difficulty": 4.0
      },
      {
        "id": "e9e5d29a9d1edea4fbdfdf7615f390b010c08f131537603f99e1757a80dc230e",
        "domain": "omni",
        "question": "Stan has a stack of 100 blocks and starts with a score of 0, and plays a game in which he iterates the following two-step procedure: (a) Stan picks a stack of blocks and splits it into 2 smaller stacks each with a positive number of blocks, say $a$ and $b$. (The order in which the new piles are placed does not matter.) (b) Stan adds the product of the two piles' sizes, $a b$, to his score. The game ends when there are only 1-block stacks left. What is the expected value of Stan's score at the end of the game?",
        "answer": "4950",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1507,
        "difficulty": 5.25
      },
      {
        "id": "dea4aa464dd0a2871f2aea40fd8b886e54e37157492e005c91ea2343feb1b5cf",
        "domain": "omni",
        "question": "Let $z$ be a non-real complex number with $z^{23}=1$. Compute $$ \\sum_{k=0}^{22} \\frac{1}{1+z^{k}+z^{2 k}} $$",
        "answer": "46 / 3",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Complex Numbers",
        "source": "HMMT_2",
        "source_index": 1615,
        "difficulty": 5.5
      },
      {
        "id": "ffb79cca3bfae78b5f47daa33ef04639590d24795d318fceefaae48e0d8184e7",
        "domain": "omni",
        "question": "Let $f$ be a polynomial with integer coefficients such that the greatest common divisor of all its coefficients is 1. For any $n \\in \\mathbb{N}, f(n)$ is a multiple of 85. Find the smallest possible degree of $f$.",
        "answer": "17",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Algebra -> Abstract Algebra -> Ring Theory",
        "source": "HMMT_2",
        "source_index": 1502,
        "difficulty": 5.5
      },
      {
        "id": "43ab07e8231456c69e4fa962cfb4bcbf79af8f9072b75970b61cb99c69ac0f63",
        "domain": "omni",
        "question": "If $x$ and $y$ are positive integers with $x+y=31$, what is the largest possible value of $x y$?",
        "answer": "240",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2676,
        "difficulty": 2.0
      },
      {
        "id": "aac495019a6848c7951307e3a15fc089aadf2923c603bf01c1eefe263d83fa58",
        "domain": "omni",
        "question": "How many positive integers \\( n \\) between 10 and 1000 have the property that the sum of the digits of \\( n \\) is 3?",
        "answer": "9",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "fermat",
        "source_index": 2944,
        "difficulty": 2.5
      },
      {
        "id": "7c7f2afed520b2e93c29508868c599458f3405a1ff75e2f50315c6e8e8e6694f",
        "domain": "omni",
        "question": "The real numbers $x, y, z, w$ satisfy $$\\begin{aligned} & 2 x+y+z+w=1 \\\\ & x+3 y+z+w=2 \\\\ & x+y+4 z+w=3 \\\\ & x+y+z+5 w=25 \\end{aligned}$$ Find the value of $w$.",
        "answer": "11/2",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 1191,
        "difficulty": 5.0
      },
      {
        "id": "fe365a5ad564bc1bd5e27573b2711efad116c7fb04e4752890e29c06a3f7abe4",
        "domain": "omni",
        "question": "A positive integer is called primer if it has a prime number of distinct prime factors. A positive integer is called primest if it has a primer number of distinct primer factors. A positive integer is called prime-minister if it has a primest number of distinct primest factors. Let $N$ be the smallest prime-minister number. Estimate $N$.",
        "answer": "378000",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2064,
        "difficulty": 5.0
      },
      {
        "id": "6c34e9e6f7ad27a509faa3139efe225bb177d83acb017933b4facbe3a0469752",
        "domain": "omni",
        "question": "The spikiness of a sequence $a_{1}, a_{2}, \\ldots, a_{n}$ of at least two real numbers is the sum $\\sum_{i=1}^{n-1}\\left|a_{i+1}-a_{i}\\right|$. Suppose $x_{1}, x_{2}, \\ldots, x_{9}$ are chosen uniformly and randomly from the interval $[0,1]$. Let $M$ be the largest possible value of the spikiness of a permutation of $x_{1}, x_{2}, \\ldots, x_{9}$. Compute the expected value of $M$.",
        "answer": "\\frac{79}{20}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1529,
        "difficulty": 5.25
      },
      {
        "id": "527c3e062287c2c1df8dce2d6984f0a1c0fecd0499fd4394951307dfb92fbad2",
        "domain": "omni",
        "question": "In-Young generates a string of $B$ zeroes and ones using the following method:\n- First, she flips a fair coin. If it lands heads, her first digit will be a 0, and if it lands tails, her first digit will be a 1.\n- For each subsequent bit, she flips an unfair coin, which lands heads with probability $A$. If the coin lands heads, she writes down the number (zero or one) different from previous digit, while if the coin lands tails, she writes down the previous digit again.\n\nWhat is the expected value of the number of zeroes in her string?",
        "answer": "2",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2171,
        "difficulty": 4.0
      },
      {
        "id": "8ca188bed13948099b236d625a157b46a8ca59e7d43ad19f1bbd588e69a51b8a",
        "domain": "omni",
        "question": "Let $x$ be a real number such that $2^{x}=3$. Determine the value of $4^{3 x+2}$.",
        "answer": "11664",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_11",
        "source_index": 1977,
        "difficulty": 3.5
      },
      {
        "id": "a2d88bb63ddffe8ac171ec89ce1f7ce139a0b9c38f4427b29efd28080901b12a",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that the polynomial $(x+1)^{n}-1$ is \"divisible by $x^{2}+1$ modulo 3\", or more precisely, either of the following equivalent conditions holds: there exist polynomials $P, Q$ with integer coefficients such that $(x+1)^{n}-1=\\left(x^{2}+1\\right) P(x)+3 Q(x)$; or more conceptually, the remainder when (the polynomial) $(x+1)^{n}-1$ is divided by (the polynomial) $x^{2}+1$ is a polynomial with (integer) coefficients all divisible by 3.",
        "answer": "8",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 633,
        "difficulty": 3.5
      },
      {
        "id": "eb338edbbb3842c1c5452bf50dba9456029e22e0d186cfe6b403738a7eeb053e",
        "domain": "omni",
        "question": "On a game show, Merble will be presented with a series of 2013 marbles, each of which is either red or blue on the outside. Each time he sees a marble, he can either keep it or pass, but cannot return to a previous marble; he receives 3 points for keeping a red marble, loses 2 points for keeping a blue marble, and gains 0 points for passing. All distributions of colors are equally likely and Merble can only see the color of his current marble. If his goal is to end with exactly one point and he plays optimally, what is the probability that he fails?",
        "answer": "\\frac{1}{2^{2012}}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1090,
        "difficulty": 4.0
      },
      {
        "id": "6b42d58ba7f6d7ee4c15ec3c6ee54d52175ad3c84d8c0d4c2c260380dda8a746",
        "domain": "omni",
        "question": "A parking lot consists of 2012 parking spots equally spaced in a line, numbered 1 through 2012. One by one, 2012 cars park in these spots under the following procedure: the first car picks from the 2012 spots uniformly randomly, and each following car picks uniformly randomly among all possible choices which maximize the minimal distance from an already parked car. What is the probability that the last car to park must choose spot 1?",
        "answer": "\\frac{1}{2062300}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1248,
        "difficulty": 4.0
      },
      {
        "id": "513c2e80063c4849668cec43f92c768bf319b333bc6bb95ed5f488498efffee1",
        "domain": "omni",
        "question": "Let $r=H_{1}$ be the answer to this problem. Given that $r$ is a nonzero real number, what is the value of $r^{4}+4 r^{3}+6 r^{2}+4 r ?$",
        "answer": "-1",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 924,
        "difficulty": 5.0
      },
      {
        "id": "5bbe50d519105295bed0735d7743388d22ccb9f537ad4e0388bdb7313a8b4058",
        "domain": "omni",
        "question": "For positive integers $x$, let $g(x)$ be the number of blocks of consecutive 1's in the binary expansion of $x$. For example, $g(19)=2$ because $19=10011_{2}$ has a block of one 1 at the beginning and a block of two 1's at the end, and $g(7)=1$ because $7=111_{2}$ only has a single block of three 1's. Compute $g(1)+g(2)+g(3)+\\cdots+g(256)$.",
        "answer": "577",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Binary Fractions -> Other",
        "source": "HMMT_2",
        "source_index": 1129,
        "difficulty": 5.0
      },
      {
        "id": "90a46310aa35812c54963ff6f7de3b4b124acb03ef20ae99404c4fd01630efb9",
        "domain": "omni",
        "question": "One bag contains 2 red marbles and 2 blue marbles. A second bag contains 2 red marbles, 2 blue marbles, and $g$ green marbles, with $g>0$. For each bag, Maria calculates the probability of randomly drawing two marbles of the same colour in two draws from that bag, without replacement. If these two probabilities are equal, what is the value of $g$?",
        "answer": "5",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 2686,
        "difficulty": 2.0
      },
      {
        "id": "69c1e7189b608dda16e4fb6843b20803a00d0f5e3a59deebf2a61fb5f5d1baed",
        "domain": "omni",
        "question": "Let $\\{a_{i}\\}_{i \\geq 0}$ be a sequence of real numbers defined by $a_{n+1}=a_{n}^{2}-\\frac{1}{2^{2020 \\cdot 2^{n}-1}}$ for $n \\geq 0$. Determine the largest value for $a_{0}$ such that $\\{a_{i}\\}_{i \\geq 0}$ is bounded.",
        "answer": "1+\\frac{1}{2^{2020}}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 725,
        "difficulty": 5.0
      },
      {
        "id": "8bbcffdafc4240655d06a2ccfea9560383b14698f56b4fa618db070d41d392a2",
        "domain": "omni",
        "question": "A ladder is leaning against a house with its lower end 15 feet from the house. When the lower end is pulled 9 feet farther from the house, the upper end slides 13 feet down. How long is the ladder (in feet)?",
        "answer": "25",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_2",
        "source_index": 3331,
        "difficulty": 4.0
      },
      {
        "id": "ab07c861ec349ba4c5e9b59565692218ee6c2df80d88e79ac8bb25d3bddf617b",
        "domain": "omni",
        "question": "John has a 1 liter bottle of pure orange juice. He pours half of the contents of the bottle into a vat, fills the bottle with water, and mixes thoroughly. He then repeats this process 9 more times. Afterwards, he pours the remaining contents of the bottle into the vat. What fraction of the liquid in the vat is now water?",
        "answer": "\\frac{5}{6}",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 3361,
        "difficulty": 3.5
      },
      {
        "id": "953554795804ea7eaf72a55c1c50e446aace7f333ea07053aea3d1bdab32bacd",
        "domain": "omni",
        "question": "A positive integer $n$ is infallible if it is possible to select $n$ vertices of a regular 100-gon so that they form a convex, non-self-intersecting $n$-gon having all equal angles. Find the sum of all infallible integers $n$ between 3 and 100, inclusive.",
        "answer": "262",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 1863,
        "difficulty": 5.0
      },
      {
        "id": "d86ecd27a1694331beb1d5d2fd5dcfed170dd44f16809d83bb8831a3e5f6847f",
        "domain": "omni",
        "question": "There are $n \\geq 2$ coins, each with a different positive integer value. Call an integer $m$ sticky if some subset of these $n$ coins have total value $m$. We call the entire set of coins a stick if all the sticky numbers form a consecutive range of integers. Compute the minimum total value of a stick across all sticks containing a coin of value 100.",
        "answer": "199",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1948,
        "difficulty": 4.5
      },
      {
        "id": "ee13b45c66ae7f83c20203e1cdf5cdcead9292b3fa9ac57300d8595cf07fef17",
        "domain": "omni",
        "question": "Suppose we keep rolling a fair 2014-sided die (whose faces are labelled 1, 2, .., 2014) until we obtain a value less than or equal to the previous roll. Let $E$ be the expected number of times we roll the die. Find the nearest integer to $100 E$.",
        "answer": "272",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1735,
        "difficulty": 4.0
      },
      {
        "id": "50d1b759509c5ca5b4cb8d5f6b2d27351df99aa55eaa896b931b4af9233b9312",
        "domain": "omni",
        "question": "Find the number of pairs of integers \\((a, b)\\) with \\(1 \\leq a<b \\leq 57\\) such that \\(a^{2}\\) has a smaller remainder than \\(b^{2}\\) when divided by 57.",
        "answer": "738",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2504,
        "difficulty": 5.25
      },
      {
        "id": "f35c81e7f378aa43a56e173753c73cbfe38ab28d8fdf6e8d5744efe116fa205d",
        "domain": "omni",
        "question": "Find the roots of the equation  $(x-a)(x-b)=(x-c)(x-d)$, if you know that $a+d=b+c=2015$ and $a \\ne c$ (numbers $a, b, c, d$ are not given).",
        "answer": "\\frac{2015}{2}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "caucasus_mathematical_olympiad",
        "source_index": 3892,
        "difficulty": 5.0
      },
      {
        "id": "0d4410352f41aabf0e75b5481999604ecc11edfaee80e8399b31d5265c62a163",
        "domain": "omni",
        "question": "A function $f(x, y)$ is linear in $x$ and in $y . f(x, y)=\\frac{1}{x y}$ for $x, y \\in\\{3,4\\}$. What is $f(5,5)$?",
        "answer": "\\frac{1}{36}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 1992,
        "difficulty": 4.5
      },
      {
        "id": "d501f38fc4cbd8fffd6aae90061404917688727df0aa8d0870a3d0ad20ed5176",
        "domain": "omni",
        "question": "Let $n$ be the maximum number of bishops that can be placed on the squares of a $6 \\times 6$ chessboard such that no two bishops are attacking each other. Let $k$ be the number of ways to put $n$ bishops on an $6 \\times 6$ chessboard such that no two bishops are attacking each other. Find $n+k$. (Two bishops are considered to be attacking each other if they lie on the same diagonal. Equivalently, if we label the squares with coordinates $(x, y)$, with $1 \\leq x, y \\leq 6$, then the bishops on $(a, b)$ and $(c, d)$ are attacking each other if and only if $|a-c|=|b-d|$.)",
        "answer": "74",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Geometry -> Plane Geometry -> Other (Chessboard problems) -> Other",
        "source": "HMMT_2",
        "source_index": 757,
        "difficulty": 5.25
      },
      {
        "id": "941ba440d12f6af795225f9db74d8617784019b9ad5f8c3f78e7451871472c38",
        "domain": "omni",
        "question": "Find the number of sets of composite numbers less than 23 that sum to 23.",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 2502,
        "difficulty": 4.5
      },
      {
        "id": "d232e4c321ca9924d8de40967d2dd61482f01dafafef1eb5508fc76c015446e8",
        "domain": "omni",
        "question": "Find the sum of all positive integers $n$ such that $1+2+\\cdots+n$ divides $15\\left[(n+1)^{2}+(n+2)^{2}+\\cdots+(2 n)^{2}\\right]$",
        "answer": "64",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers; Mathematics -> Number Theory -> Divisibility -> Other; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2008,
        "difficulty": 4.5
      },
      {
        "id": "1b371c8c4a34576e383a242d93caadb678c73e6d1b7d12c2cdbd12daa4277b59",
        "domain": "omni",
        "question": "Ten positive integers are arranged around a circle. Each number is one more than the greatest common divisor of its two neighbors. What is the sum of the ten numbers?",
        "answer": "28",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_2",
        "source_index": 1352,
        "difficulty": 5.0
      },
      {
        "id": "9739147ebe46f33990a7ef53c3d4fbb10b01ff824af28dd9595e4ac4f90e180a",
        "domain": "omni",
        "question": "In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \\perp BC, BE \\perp CA, CF \\perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \\perp EF$ and $HQ \\perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.",
        "answer": "1",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "usa_team_selection_test",
        "source_index": 1,
        "difficulty": 7.0
      },
      {
        "id": "698fb4d28249e2e1ab5f1d554da7dfe1d7ed2428efc422ffe51bb54aff20704f",
        "domain": "omni",
        "question": "Let $f$ be a function that takes in a triple of integers and outputs a real number. Suppose that $f$ satisfies the equations $f(a, b, c) =\\frac{f(a+1, b, c)+f(a-1, b, c)}{2}$, $f(a, b, c) =\\frac{f(a, b+1, c)+f(a, b-1, c)}{2}$, $f(a, b, c) =\\frac{f(a, b, c+1)+f(a, b, c-1)}{2}$ for all integers $a, b, c$. What is the minimum number of triples at which we need to evaluate $f$ in order to know its value everywhere?",
        "answer": "8",
        "topic": "Mathematics -> Precalculus -> Functions; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 1666,
        "difficulty": 4.5
      },
      {
        "id": "067dd0b882038ada75b323bc2bc4d55272134478e9dd5f263c8ca8445bc096ea",
        "domain": "omni",
        "question": "Define $x \\star y=\\frac{\\sqrt{x^{2}+3 x y+y^{2}-2 x-2 y+4}}{x y+4}$. Compute $$((\\cdots((2007 \\star 2006) \\star 2005) \\star \\cdots) \\star 1)$$",
        "answer": "\\frac{\\sqrt{15}}{9}",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "HMMT_2",
        "source_index": 1813,
        "difficulty": 5.0
      },
      {
        "id": "df760f55f8aee7146c7a61bb34474bf59d99b0d99a1f91613824490762220f32",
        "domain": "omni",
        "question": "If $x y=5$ and $x^{2}+y^{2}=21$, compute $x^{4}+y^{4}$.",
        "answer": "391",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 417,
        "difficulty": 3.5
      },
      {
        "id": "80ea7e2e596f6f851334fa85d38601ade8718b7e5803d0389bfff96977a3ce9c",
        "domain": "omni",
        "question": "What is the value of the expression $2 \\times 3 + 2 \\times 3$?",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2938,
        "difficulty": 1.0
      },
      {
        "id": "aba352fcdb5f4586365478d7908bffba9e15f2f6b00b1524aba25ac7ffc5c04c",
        "domain": "omni",
        "question": "Let $A, E, H, L, T$, and $V$ be chosen independently and at random from the set $\\left\\{0, \\frac{1}{2}, 1\\right\\}$. Compute the probability that $\\lfloor T \\cdot H \\cdot E\\rfloor=L \\cdot A \\cdot V \\cdot A$.",
        "answer": "\\frac{55}{81}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1677,
        "difficulty": 3.5
      },
      {
        "id": "411da9f4cd7682f47d995b09d27b746132d33349e2f895cb78793844a82e2582",
        "domain": "omni",
        "question": "Find the smallest positive integer $n$ such that, if there are initially $2n$ townspeople and 1 goon, then the probability the townspeople win is greater than $50\\%$.",
        "answer": "3",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_11",
        "source_index": 2067,
        "difficulty": 5.0
      },
      {
        "id": "6ffd0704093ef7392a59d8b2178b6a844ba64085c3b1e6fa0849c42a73ab2bf7",
        "domain": "omni",
        "question": "Given a set $A$ with 10 elements, find the number of consistent 2-configurations of $A$ of order 2 with exactly 2 cells.",
        "answer": "99144",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1294,
        "difficulty": 4.5
      },
      {
        "id": "5a4ef1c14cd1d7afb5b7382f1f443323c9d2c616fcc598205265a7b054633db8",
        "domain": "omni",
        "question": "In a $k \\times k$ chessboard, a set $S$ of 25 cells that are in a $5 \\times 5$ square is chosen uniformly at random. The probability that there are more black squares than white squares in $S$ is $48 \\%$. Find $k$.",
        "answer": "9",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2520,
        "difficulty": 5.0
      },
      {
        "id": "c65a41fdd9b56b2293f72ae56d83a0f95cc703b3161d7d4ccb261c8cc9360f6f",
        "domain": "omni",
        "question": "If $wxyz$ is a four-digit positive integer with $w \\neq 0$, the layer sum of this integer equals $wxyz + xyz + yz + z$. If the layer sum of $wxyz$ equals 2014, what is the value of $w + x + y + z$?",
        "answer": "13",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3130,
        "difficulty": 2.0
      },
      {
        "id": "d0577e83caf400d3e4a4f6f65846a919ade3282c234580dc9ef00a17486428cd",
        "domain": "omni",
        "question": "A 5 by 5 grid of unit squares is partitioned into 5 pairwise incongruent rectangles with sides lying on the gridlines. Find the maximum possible value of the product of their areas.",
        "answer": "2304",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Area",
        "source": "HMMT_2",
        "source_index": 864,
        "difficulty": 4.5
      },
      {
        "id": "226ff42b79c7e7e834a2d5155f4903f8553501b76f97a4d7d3d9939d09eb1d3f",
        "domain": "omni",
        "question": "Find the sum of all real numbers $x$ for which $$\\lfloor\\lfloor\\cdots\\lfloor\\lfloor\\lfloor x\\rfloor+x\\rfloor+x\\rfloor \\cdots\\rfloor+x\\rfloor=2017 \\text { and }\\{\\{\\cdots\\{\\{\\{x\\}+x\\}+x\\} \\cdots\\}+x\\}=\\frac{1}{2017}$$ where there are $2017 x$ 's in both equations. ( $\\lfloor x\\rfloor$ is the integer part of $x$, and $\\{x\\}$ is the fractional part of $x$.) Express your sum as a mixed number.",
        "answer": "3025 \\frac{1}{2017}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2555,
        "difficulty": 4.5
      },
      {
        "id": "bc33162628f395c7bea23a5c48b8adbf78c6c725b064337187cfadf17664da75",
        "domain": "omni",
        "question": "Suppose that $\\sqrt{\\frac{1}{2} \\times \\frac{2}{3} \\times \\frac{3}{4} \\times \\frac{4}{5} \\times \\cdots \\times \\frac{n-1}{n}} = \\frac{1}{8}$. What is the value of $n$?",
        "answer": "64",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "cayley",
        "source_index": 3039,
        "difficulty": 1.5
      },
      {
        "id": "3f7f9f5387362399025f70562ab514b27b466b1897cd8596bc31054a7daeaf7e",
        "domain": "omni",
        "question": "Compute the smallest positive integer that does not appear in any problem statement on any round at HMMT November 2023.",
        "answer": "22",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1858,
        "difficulty": 3.0
      },
      {
        "id": "93eeeaf3290e97ae389c0c8b538809fdccdedb4ef52b332b451075634e6bff5a",
        "domain": "omni",
        "question": "What is the smallest positive integer $t$ such that there exist integers $x_1,x_2,\\ldots,x_t$ with  \\[x^3_1+x^3_2+\\,\\ldots\\,+x^3_t=2002^{2002}\\,?\\]",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "imo_shortlist",
        "source_index": 3982,
        "difficulty": 9.0
      },
      {
        "id": "fcd1a5243418f7fb4747c36f2fdf1e684fd291016027045e1a205b8b6227ac7e",
        "domain": "omni",
        "question": "The numbers $1-10$ are written in a circle randomly. Find the expected number of numbers which are at least 2 larger than an adjacent number.",
        "answer": "\\frac{17}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1839,
        "difficulty": 4.5
      },
      {
        "id": "793fd9f1f0fc1ba870aa936dc223ddb775d22ad576d34a180d5cf8b77a6a1477",
        "domain": "omni",
        "question": "The line with equation $y=2x-6$ is translated upwards by 4 units. What is the $x$-intercept of the resulting line?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions",
        "source": "fermat",
        "source_index": 3017,
        "difficulty": 2.0
      },
      {
        "id": "219cd3791545e1c98c0e651de82aa93dea8d4e5d27fdd7c4720d5eb03f865904",
        "domain": "omni",
        "question": "Let $S=\\{1,2, \\ldots, 2021\\}$, and let $\\mathcal{F}$ denote the set of functions $f: S \\rightarrow S$. For a function $f \\in \\mathcal{F}$, let $$T_{f}=\\left\\{f^{2021}(s): s \\in S\\right\\}$$ where $f^{2021}(s)$ denotes $f(f(\\cdots(f(s)) \\cdots))$ with 2021 copies of $f$. Compute the remainder when $$\\sum_{f \\in \\mathcal{F}}\\left|T_{f}\\right|$$ is divided by the prime 2017, where the sum is over all functions $f$ in $\\mathcal{F}$.",
        "answer": "255",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics; Mathematics -> Algebra -> Polynomial Operations -> Other",
        "source": "HMMT_2",
        "source_index": 1172,
        "difficulty": 5.0
      },
      {
        "id": "f4115d9b1ed4dcf79210bd1f41adfd3c2329e06fbb5ff8520a0b104bbca648c7",
        "domain": "omni",
        "question": "A rectangle with dimensions 100 cm by 150 cm is tilted so that one corner is 20 cm above a horizontal line, as shown. To the nearest centimetre, the height of vertex $Z$ above the horizontal line is $(100+x) \\mathrm{cm}$. What is the value of $x$?",
        "answer": "67",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "fermat",
        "source_index": 2946,
        "difficulty": 2.5
      },
      {
        "id": "4036376b4154ab54689074d501f5f233fad31479efdcd9a0afbcd6a4e316e6e0",
        "domain": "omni",
        "question": "What is the smallest integer greater than 10 such that the sum of the digits in its base 17 representation is equal to the sum of the digits in its base 10 representation?",
        "answer": "153",
        "topic": "Mathematics -> Number Theory -> Other; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 2538,
        "difficulty": 3.5
      },
      {
        "id": "eeaa84a0ebf85cb83c6bb0c64e312da46633e1f74405c8fc3821d083d1d332c3",
        "domain": "omni",
        "question": "In a group of people, there are 13 who like apples, 9 who like blueberries, 15 who like cantaloupe, and 6 who like dates. (A person can like more than 1 kind of fruit.) Each person who likes blueberries also likes exactly one of apples and cantaloupe. Each person who likes cantaloupe also likes exactly one of blueberries and dates. Find the minimum possible number of people in the group.",
        "answer": "22",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 2333,
        "difficulty": 4.0
      },
      {
        "id": "4940e878ae8221b95ab4c0b922d1468111fbccdd5f4f861afcf37a24e19299f0",
        "domain": "omni",
        "question": "There are three video game systems: the Paystation, the WHAT, and the ZBoz2 \\pi, and none of these systems will play games for the other systems. Uncle Riemann has three nephews: Bernoulli, Galois, and Dirac. Bernoulli owns a Paystation and a WHAT, Galois owns a WHAT and a ZBoz2 \\pi, and Dirac owns a ZBoz2 \\pi and a Paystation. A store sells 4 different games for the Paystation, 6 different games for the WHAT, and 10 different games for the ZBoz2 \\pi. Uncle Riemann does not understand the difference between the systems, so he walks into the store and buys 3 random games (not necessarily distinct) and randomly hands them to his nephews. What is the probability that each nephew receives a game he can play?",
        "answer": "\\frac{7}{25}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1532,
        "difficulty": 3.5
      },
      {
        "id": "c12a87cc36df0412c788886e8f7edbcd4833f4dd4d3eb78833fc5affaa9a83bc",
        "domain": "omni",
        "question": "What is the smallest positive integer $n$ which cannot be written in any of the following forms? - $n=1+2+\\cdots+k$ for a positive integer $k$. - $n=p^{k}$ for a prime number $p$ and integer $k$ - $n=p+1$ for a prime number $p$. - $n=p q$ for some distinct prime numbers $p$ and $q$",
        "answer": "40",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_11",
        "source_index": 1721,
        "difficulty": 5.0
      },
      {
        "id": "83c2148ab80808e5ce29c13de12e8f10ff4ae8b2ad09886943fcc978b9ad91cd",
        "domain": "omni",
        "question": "A function $f$ satisfies, for all nonnegative integers $x$ and $y$: - $f(0, x)=f(x, 0)=x$ - If $x \\geq y \\geq 0, f(x, y)=f(x-y, y)+1$ - If $y \\geq x \\geq 0, f(x, y)=f(x, y-x)+1$ Find the maximum value of $f$ over $0 \\leq x, y \\leq 100$.",
        "answer": "101",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 1802,
        "difficulty": 5.25
      },
      {
        "id": "365161c6cd5477f6d53ade1028cacf699698c94cad3e0da130acb8f81a860cf9",
        "domain": "omni",
        "question": "The sum of five consecutive odd integers is 125. What is the smallest of these integers?",
        "answer": "21",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2711,
        "difficulty": 2.0
      },
      {
        "id": "edb69a9dd295bc6765c03c67c69b3e3be8f17db8924c37276d014362c0661f93",
        "domain": "omni",
        "question": "A sequence is defined by $A_{0}=0, A_{1}=1, A_{2}=2$, and, for integers $n \\geq 3$, $$A_{n}=\\frac{A_{n-1}+A_{n-2}+A_{n-3}}{3}+\\frac{1}{n^{4}-n^{2}}$$ Compute $\\lim _{N \\rightarrow \\infty} A_{N}$.",
        "answer": "\\frac{13}{6}-\\frac{\\pi^{2}}{12}",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Partial Fractions -> Other",
        "source": "HMMT_2",
        "source_index": 572,
        "difficulty": 4.5
      },
      {
        "id": "d96c1ebc42bb75beaca3535e0a66f00bcd5b9c92ea0ba922306fa3ff8b8aa4b8",
        "domain": "omni",
        "question": "Reduce the number $\\sqrt[3]{2+\\sqrt{5}}+\\sqrt[3]{2-\\sqrt{5}}$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 3337,
        "difficulty": 4.0
      },
      {
        "id": "6afbf733cbed979bb936ecfba04f3dec416394453f7a06eb4055abb2602803fb",
        "domain": "omni",
        "question": "The expression $\\lfloor x\\rfloor$ denotes the greatest integer less than or equal to $x$. Find the value of $$\\left\\lfloor\\frac{2002!}{2001!+2000!+1999!+\\cdots+1!}\\right\\rfloor.$$",
        "answer": "2000",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Other",
        "source": "HMMT_2",
        "source_index": 816,
        "difficulty": 4.0
      },
      {
        "id": "cf00813fa77401adaf5cd69676e6faeb216589e1009ec86109629cf40bed7381",
        "domain": "omni",
        "question": "In the diagram below, how many distinct paths are there from January 1 to December 31, moving from one adjacent dot to the next either to the right, down, or diagonally down to the right?",
        "answer": "372",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1501,
        "difficulty": 4.0
      },
      {
        "id": "f145b40b3df8c118a849c7c69f4151173cbddd97cf644233b49ec699bcf36a6d",
        "domain": "omni",
        "question": "Let $P$ and $A$ denote the perimeter and area respectively of a right triangle with relatively prime integer side-lengths. Find the largest possible integral value of $\\frac{P^{2}}{A}$.",
        "answer": "45",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 466,
        "difficulty": 5.0
      },
      {
        "id": "2552ce1cab31f0a9b05ae99e28bc8608958bf5f5a99a21372c13f58b6ede118d",
        "domain": "omni",
        "question": "At a recent math contest, Evan was asked to find $2^{2016}(\\bmod p)$ for a given prime number $p$ with $100<p<500$. Evan has forgotten what the prime $p$ was, but still remembers how he solved it: - Evan first tried taking 2016 modulo $p-1$, but got a value $e$ larger than 100. - However, Evan noted that $e-\\frac{1}{2}(p-1)=21$, and then realized the answer was $-2^{21}(\\bmod p)$. What was the prime $p$?",
        "answer": "211",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 460,
        "difficulty": 5.0
      },
      {
        "id": "4d60f888b7f23bee7b348bb2d7ae59451fb4d0af577be4f6eba075be2c90462a",
        "domain": "omni",
        "question": "The average of $a, b$ and $c$ is 16. The average of $c, d$ and $e$ is 26. The average of $a, b, c, d$, and $e$ is 20. What is the value of $c$?",
        "answer": "26",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 2734,
        "difficulty": 2.5
      },
      {
        "id": "90bc3ad19a4e41f67fd876e74b0e64fd4e2a399f772c2dc5963501a84981f5f1",
        "domain": "omni",
        "question": "Evaluate the infinite sum $\\sum_{n=1}^{\\infty} \\frac{n}{n^{4}+4}$.",
        "answer": "\\frac{3}{8}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 486,
        "difficulty": 5.25
      },
      {
        "id": "c1344252f680514a3558f246dab6802857bd6a69d5954058d3602f6426baf3f0",
        "domain": "omni",
        "question": "How many integers between 100 and 300 are multiples of both 5 and 7, but are not multiples of 10?",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2966,
        "difficulty": 2.0
      },
      {
        "id": "aaa5e080daaf6ac03b3585088621a457d6300617fcebe756c7f514e499245acc",
        "domain": "omni",
        "question": "Estimate the number of positive integers $n \\leq 10^{6}$ such that $n^{2}+1$ has a prime factor greater than $n$. Submit a positive integer $E$. If the correct answer is $A$, you will receive $\\max \\left(0,\\left\\lfloor 20 \\cdot \\min \\left(\\frac{E}{A}, \\frac{10^{6}-E}{10^{6}-A}\\right)^{5}+0.5\\right\\rfloor\\right)$ points.",
        "answer": "757575",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Calculus -> Integral Calculus -> Techniques of Integration -> Single-variable",
        "source": "HMMT_2",
        "source_index": 887,
        "difficulty": 5.25
      },
      {
        "id": "c7bc4c0702d95328a6df0619aade1b7d9a8634054ad1caefc50375f0533c0c5b",
        "domain": "omni",
        "question": "Two reals \\( x \\) and \\( y \\) are such that \\( x-y=4 \\) and \\( x^{3}-y^{3}=28 \\). Compute \\( x y \\).",
        "answer": "-3",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1485,
        "difficulty": 4.0
      },
      {
        "id": "349e845e8bbab04640d6a9b009a3709ea1607401e401fbdb949ab379716c0151",
        "domain": "omni",
        "question": "An infinite sequence of real numbers $a_{1}, a_{2}, \\ldots$ satisfies the recurrence $$a_{n+3}=a_{n+2}-2 a_{n+1}+a_{n}$$ for every positive integer $n$. Given that $a_{1}=a_{3}=1$ and $a_{98}=a_{99}$, compute $a_{1}+a_{2}+\\cdots+a_{100}$.",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Algebra -> Sequences and Series",
        "source": "HMMT_2",
        "source_index": 1462,
        "difficulty": 5.0
      },
      {
        "id": "04c159828d0fd323c1a1f8651cbe7c6841d0a712415ba4d599b20f27ea52e9b4",
        "domain": "omni",
        "question": "If $x=2$, what is the value of $4x^2 - 3x^2$?",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2906,
        "difficulty": 1.0
      },
      {
        "id": "dc7d62245d007ae9221c76f59342d84c5f277743dfd3cd7f51ba86c2ae94b89b",
        "domain": "omni",
        "question": "Let $a, b, c, n$ be positive real numbers such that $\\frac{a+b}{a}=3, \\frac{b+c}{b}=4$, and $\\frac{c+a}{c}=n$. Find $n$.",
        "answer": "\\frac{7}{6}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1893,
        "difficulty": 5.0
      },
      {
        "id": "18ac5dfc302bd4d2282041ebb97b31b98fd1f59b1c1fff6db4faabfbe9bab47d",
        "domain": "omni",
        "question": "Bob Barker went back to school for a PhD in math, and decided to raise the intellectual level of The Price is Right by having contestants guess how many objects exist of a certain type, without going over. The number of points you will get is the percentage of the correct answer, divided by 10, with no points for going over (i.e. a maximum of 10 points). Let's see the first object for our contestants...a table of shape (5,4,3,2,1) is an arrangement of the integers 1 through 15 with five numbers in the top row, four in the next, three in the next, two in the next, and one in the last, such that each row and each column is increasing (from left to right, and top to bottom, respectively). For instance: \\begin{tabular}{lcccc} 1 & 2 & 3 & 4 & 5 \\\\ 6 & 7 & 8 & 9 & \\\\ 10 & 11 & 12 & & \\\\ 13 & 14 & & & \\\\ 15 & & & & \\end{tabular} is one table. How many tables are there?",
        "answer": "292864",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 797,
        "difficulty": 5.0
      },
      {
        "id": "4ccd85d3bd5a34c86443bfac9366470969dae68eb4c493ea5c7546418d91dd1b",
        "domain": "omni",
        "question": "For each positive integer $n$, there is a circle around the origin with radius $n$. Rainbow Dash starts off somewhere on the plane, but not on a circle. She takes off in some direction in a straight path. She moves \\frac{\\sqrt{5}}{5}$ units before crossing a circle, then \\sqrt{5}$ units, then \\frac{3 \\sqrt{5}}{5}$ units. What distance will she travel before she crosses another circle?",
        "answer": "\\frac{2 \\sqrt{170}-9 \\sqrt{5}}{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 3326,
        "difficulty": 5.0
      },
      {
        "id": "4db2842785d3f3deceb7d73ab32c22803fb3a3486d0bcfe2a0f17ef6543d4dbe",
        "domain": "omni",
        "question": "Let the circles $k_1$ and $k_2$ intersect at two points $A$ and $B$ , and let $t$ be a common tangent of $k_1$ and $k_2$ that touches $k_1$ and $k_2$ at $M$ and $N$ respectively. If $t\\perp AM$ and $MN=2AM$ , evaluate the angle $NMB$ .",
        "answer": "\\[\n\\boxed{\\frac{\\pi}{4}}\n\\]",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles",
        "source": "jbmo",
        "source_index": 178,
        "difficulty": 4.5
      },
      {
        "id": "3f887c9207f71a963e25cf5eca1598a0e5a10b1e3c199c6a16e6466851353dfb",
        "domain": "omni",
        "question": "Daniel and Scott are playing a game where a player wins as soon as he has two points more than his opponent. Both players start at par, and points are earned one at a time. If Daniel has a $60 \\%$ chance of winning each point, what is the probability that he will win the game?",
        "answer": "9 / 13",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 1468,
        "difficulty": 5.0
      },
      {
        "id": "84014d22302ce1a976eb1f23c68d344ca46df7dde560bc42eb56d4510da326d5",
        "domain": "omni",
        "question": "A regular $n$-gon $P_{1} P_{2} \\ldots P_{n}$ satisfies $\\angle P_{1} P_{7} P_{8}=178^{\\circ}$. Compute $n$.",
        "answer": "630",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2224,
        "difficulty": 4.0
      },
      {
        "id": "78a71f2f41ba0017477ea4c756a9cefad5ec2279881e598ef072eb657a881d57",
        "domain": "omni",
        "question": "Trodgor the dragon is burning down a village consisting of 90 cottages. At time $t=0$ an angry peasant arises from each cottage, and every 8 minutes (480 seconds) thereafter another angry peasant spontaneously generates from each non-burned cottage. It takes Trodgor 5 seconds to either burn a peasant or to burn a cottage, but Trodgor cannot begin burning cottages until all the peasants around him have been burned. How many seconds does it take Trodgor to burn down the entire village?",
        "answer": "1920",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_2",
        "source_index": 484,
        "difficulty": 5.0
      },
      {
        "id": "e2587bc8a6200cd1a42255f08eed53e77e3029e1515fdccf68bb949698051687",
        "domain": "omni",
        "question": "Find the number of solutions to the equation $x+y+z=525$ where $x$ is a multiple of 7, $y$ is a multiple of 5, and $z$ is a multiple of 3.",
        "answer": "21",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 2415,
        "difficulty": 4.0
      },
      {
        "id": "0db331c9c65fb9a161fb75dbfb9a61a0c0aef9e0bc359316b779a2f13e1f96f7",
        "domain": "omni",
        "question": "Let $\\mathbb{R}$ be the set of real numbers. Let $f: \\mathbb{R} \\rightarrow \\mathbb{R}$ be a function such that for all real numbers $x$ and $y$, we have $$f\\left(x^{2}\\right)+f\\left(y^{2}\\right)=f(x+y)^{2}-2 x y$$ Let $S=\\sum_{n=-2019}^{2019} f(n)$. Determine the number of possible values of $S$.",
        "answer": "2039191",
        "topic": "Mathematics -> Precalculus -> Functions; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 993,
        "difficulty": 5.25
      },
      {
        "id": "89ba05ec1884d7f85069e8880505819d7fbfca548e40fdaaf0a67147f756c616",
        "domain": "omni",
        "question": "In the addition problem shown, $m, n, p$, and $q$ represent positive digits. What is the value of $m+n+p+q$?",
        "answer": "24",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "cayley",
        "source_index": 2865,
        "difficulty": 1.5
      },
      {
        "id": "d8f5b9257d183f9e5b80e772130728040e946d55c4e6a2c2835ab39c68e788d6",
        "domain": "omni",
        "question": "The International Mathematical Olympiad is being organized in Japan, where a folklore belief is that the number $4$ brings bad luck. The opening ceremony takes place at the Grand Theatre where each row has the capacity of $55$ seats. What is the maximum number of contestants that can be seated in a single row with the restriction that no two of them are $4$ seats apart (so that bad luck during the competition is avoided)?",
        "answer": "30",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "balkan_mo_shortlist",
        "source_index": 3992,
        "difficulty": 6.0
      },
      {
        "id": "e54edbc751cc7ce17d745a9132d16d5c830ed86fec4a5b4838afe754284c888a",
        "domain": "omni",
        "question": "How many non-empty subsets of $\\{1,2,3,4,5,6,7,8\\}$ have exactly $k$ elements and do not contain the element $k$ for some $k=1,2, \\ldots, 8$.",
        "answer": "127",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 937,
        "difficulty": 4.0
      },
      {
        "id": "926774a716739ebaab889cd4141e64e4dfcd37ea60cbf4b73dd755e46c604f8b",
        "domain": "omni",
        "question": "Zlatan has 2017 socks of various colours. He wants to proudly display one sock of each of the colours, and he counts that there are $N$ ways to select socks from his collection for display. Given this information, what is the maximum value of $N$?",
        "answer": "3^{671} \\cdot 4",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2085,
        "difficulty": 5.0
      },
      {
        "id": "a0025ee4097746514a1cef368f284426cc641acb6549d3f973a65816b9efeed3",
        "domain": "omni",
        "question": "Equilateral triangle $ABC$ has circumcircle $\\Omega$. Points $D$ and $E$ are chosen on minor arcs $AB$ and $AC$ of $\\Omega$ respectively such that $BC=DE$. Given that triangle $ABE$ has area 3 and triangle $ACD$ has area 4, find the area of triangle $ABC$.",
        "answer": "\\frac{37}{7}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2145,
        "difficulty": 4.5
      },
      {
        "id": "cc680aee8b528d0825fdf52ed80b752efb14bed918bb25c2747d0cfe15c73487",
        "domain": "omni",
        "question": "Will stands at a point \\(P\\) on the edge of a circular room with perfectly reflective walls. He shines two laser pointers into the room, forming angles of \\(n^{\\circ}\\) and \\((n+1)^{\\circ}\\) with the tangent at \\(P\\), where \\(n\\) is a positive integer less than 90. The lasers reflect off of the walls, illuminating the points they hit on the walls, until they reach \\(P\\) again. (\\(P\\) is also illuminated at the end.) What is the minimum possible number of illuminated points on the walls of the room?",
        "answer": "28",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Number Theory -> Greatest Common Divisors (GCD)",
        "source": "HMMT_11",
        "source_index": 2163,
        "difficulty": 5.5
      },
      {
        "id": "1b886c619f847eb9f7b404cb37b4af634e1b3efcfd20ee8e31c41b809a9966ec",
        "domain": "omni",
        "question": "In $\\triangle A B C, A B=2019, B C=2020$, and $C A=2021$. Yannick draws three regular $n$-gons in the plane of $\\triangle A B C$ so that each $n$-gon shares a side with a distinct side of $\\triangle A B C$ and no two of the $n$-gons overlap. What is the maximum possible value of $n$?",
        "answer": "11",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2252,
        "difficulty": 4.0
      },
      {
        "id": "a3d8a98b8400026fdfde2dc848bac9d38efe6bc236672f4fc2157c2b6982e59f",
        "domain": "omni",
        "question": "Let $X Y Z$ be an equilateral triangle, and let $K, L, M$ be points on sides $X Y, Y Z, Z X$, respectively, such that $X K / K Y=B, Y L / L Z=1 / C$, and $Z M / M X=1$. Determine the ratio of the area of triangle $K L M$ to the area of triangle $X Y Z$.",
        "answer": "$\\frac{1}{5}$",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2059,
        "difficulty": 4.5
      },
      {
        "id": "dc94e587a2822a24ff06d640cd321f7de95abc5c9edc5dc221b224e363733901",
        "domain": "omni",
        "question": "Evaluate the sum $$\\frac{1}{2\\lfloor\\sqrt{1}\\rfloor+1}+\\frac{1}{2\\lfloor\\sqrt{2}\\rfloor+1}+\\frac{1}{2\\lfloor\\sqrt{3}\\rfloor+1}+\\cdots+\\frac{1}{2\\lfloor\\sqrt{100}\\rfloor+1}$$",
        "answer": "190/21",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Fractions",
        "source": "HMMT_2",
        "source_index": 1303,
        "difficulty": 5.0
      },
      {
        "id": "03b12899050ce8caf7df1c00f90d98c287b936b0f3c2c7f82404abe86750dcc3",
        "domain": "omni",
        "question": "In a chess-playing club, some of the players take lessons from other players. It is possible (but not necessary) for two players both to take lessons from each other. It so happens that for any three distinct members of the club, $A, B$, and $C$, exactly one of the following three statements is true: $A$ takes lessons from $B ; B$ takes lessons from $C ; C$ takes lessons from $A$. What is the largest number of players there can be?",
        "answer": "4",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 987,
        "difficulty": 5.25
      },
      {
        "id": "cc54a06e99512c32b95a69a702e5dbc5530b572bcb509d8658e2218b42691d52",
        "domain": "omni",
        "question": "Find the number of ordered 2012-tuples of integers $\\left(x_{1}, x_{2}, \\ldots, x_{2012}\\right)$, with each integer between 0 and 2011 inclusive, such that the sum $x_{1}+2 x_{2}+3 x_{3}+\\cdots+2012 x_{2012}$ is divisible by 2012.",
        "answer": "2012^{2011}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 2184,
        "difficulty": 4.0
      },
      {
        "id": "31c3bc145bab01049ce4b820d4fef88dcc9c453bd389635b747fd4bf4ce27a06",
        "domain": "omni",
        "question": "A deck of 100 cards is labeled $1,2, \\ldots, 100$ from top to bottom. The top two cards are drawn; one of them is discarded at random, and the other is inserted back at the bottom of the deck. This process is repeated until only one card remains in the deck. Compute the expected value of the label of the remaining card.",
        "answer": "\\frac{467}{8}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 766,
        "difficulty": 4.5
      },
      {
        "id": "0bc39c04648e492e26def269c1d24f85919fa54121f337a4e26b355513a9141e",
        "domain": "omni",
        "question": "Find the sum of the digits of \\(11 \\cdot 101 \\cdot 111 \\cdot 110011\\).",
        "answer": "48",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1566,
        "difficulty": 3.5
      },
      {
        "id": "b3420db9ce3570e44acf4fcdb87ec890c8de0b0e40d240b4dca2ea332001fc88",
        "domain": "omni",
        "question": "Let $P(x)=x^{3}+a x^{2}+b x+2015$ be a polynomial all of whose roots are integers. Given that $P(x) \\geq 0$ for all $x \\geq 0$, find the sum of all possible values of $P(-1)$.",
        "answer": "9496",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 791,
        "difficulty": 6.0
      },
      {
        "id": "9bf2ad6b41fea82810b930d2937eead27f0d93ccdc33127738123f1fee7d22bd",
        "domain": "omni",
        "question": "Lil Wayne, the rain god, determines the weather. If Lil Wayne makes it rain on any given day, the probability that he makes it rain the next day is $75 \\%$. If Lil Wayne doesn't make it rain on one day, the probability that he makes it rain the next day is $25 \\%$. He decides not to make it rain today. Find the smallest positive integer $n$ such that the probability that Lil Wayne makes it rain $n$ days from today is greater than $49.9 \\%$.",
        "answer": "9",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "HMMT_11",
        "source_index": 2317,
        "difficulty": 4.0
      },
      {
        "id": "6c8ad6ce013127a96cdaa93b024be6d18111e8ef3733475aea6f25910d66e7e6",
        "domain": "omni",
        "question": "Given that the area of a rectangle is 192 and its length is 24, what is the perimeter of the rectangle?",
        "answer": "64",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations; Mathematics -> Geometry -> Plane Geometry -> Perimeter",
        "source": "fermat",
        "source_index": 2908,
        "difficulty": 2.5
      },
      {
        "id": "65b82d47c5441151dbafbc80e68278e553e3acdd8617633dec3fb1b3a4c1dc3c",
        "domain": "omni",
        "question": "Determine the value of $$1 \\cdot 2-2 \\cdot 3+3 \\cdot 4-4 \\cdot 5+\\cdots+2001 \\cdot 2002$$",
        "answer": "2004002",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Arithmetic Sequences -> Other",
        "source": "HMMT_2",
        "source_index": 844,
        "difficulty": 5.0
      },
      {
        "id": "227226a9a420ee2fe74841cb79215f4ac2b737410981f715cb5f37bca9c3e333",
        "domain": "omni",
        "question": "If $(2)(3)(4) = 6x$, what is the value of $x$?",
        "answer": "4",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "cayley",
        "source_index": 3065,
        "difficulty": 1.5
      },
      {
        "id": "f2039d30ce0da4d2e19c94d721a173e3868cb194950aea555168613065fbb436",
        "domain": "omni",
        "question": "The Evil League of Evil is plotting to poison the city's water supply. They plan to set out from their headquarters at $(5,1)$ and put poison in two pipes, one along the line $y=x$ and one along the line $x=7$. However, they need to get the job done quickly before Captain Hammer catches them. What's the shortest distance they can travel to visit both pipes and then return to their headquarters?",
        "answer": "4 \\sqrt{5}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles",
        "source": "HMMT_2",
        "source_index": 901,
        "difficulty": 5.25
      },
      {
        "id": "41db8f5b730902a75b278238ba925ab482df8e804c66a445fcf2296daedd7daf",
        "domain": "omni",
        "question": "What is the sum of all numbers $q$ which can be written in the form $q=\\frac{a}{b}$ where $a$ and $b$ are positive integers with $b \\leq 10$ and for which there are exactly 19 integers $n$ that satisfy $\\sqrt{q}<n<q$?",
        "answer": "777.5",
        "topic": "Mathematics -> Algebra -> Algebra -> Algebraic Expressions; Mathematics -> Number Theory -> Prime Numbers",
        "source": "cayley",
        "source_index": 3140,
        "difficulty": 2.5
      },
      {
        "id": "73eba65f34cf7e9ac87f1bc88029792a0c004a5c3a6361f503eee7d7bf523608",
        "domain": "omni",
        "question": "In how many ways can you fill a $3 \\times 3$ table with the numbers 1 through 9 (each used once) such that all pairs of adjacent numbers (sharing one side) are relatively prime?",
        "answer": "2016",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 2596,
        "difficulty": 5.25
      },
      {
        "id": "d9f28d687878e96a8d1e4ef3070ff02aad160bebc3fe20904d4170a8412a7994",
        "domain": "omni",
        "question": "If $x+2 y-3 z=7$ and $2 x-y+2 z=6$, determine $8 x+y$.",
        "answer": "32",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "HMMT_11",
        "source_index": 1859,
        "difficulty": 3.5
      },
      {
        "id": "3a6cab79c11b6b536c669919f31309d95794a5e843c2a835082b83b6f1afbf4d",
        "domain": "omni",
        "question": "Massachusetts Avenue is ten blocks long. One boy and one girl live on each block. They want to form friendships such that each boy is friends with exactly one girl and vice versa. Nobody wants a friend living more than one block away (but they may be on the same block). How many pairings are possible?",
        "answer": "89",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 955,
        "difficulty": 5.0
      },
      {
        "id": "747bf5e6fbf1c50e6b44888502d2e2073d9bcfa41be7a11a3ff96dfa3b636a62",
        "domain": "omni",
        "question": "Jack, Jill, and John play a game in which each randomly picks and then replaces a card from a standard 52 card deck, until a spades card is drawn. What is the probability that Jill draws the spade? (Jack, Jill, and John draw in that order, and the game repeats if no spade is drawn.)",
        "answer": "\\frac{12}{37}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1334,
        "difficulty": 4.0
      },
      {
        "id": "66a6731f5ddaaf0cc477cc2b4f798f6fe24935c616f467fe3389dc084da9414f",
        "domain": "omni",
        "question": "Two standard six-sided dice are rolled. What is the probability that the product of the two numbers rolled is 12?",
        "answer": "\\frac{4}{36}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "cayley",
        "source_index": 2719,
        "difficulty": 1.5
      },
      {
        "id": "db60a417616944657458a8afd0e9a275d7fee0fa524131a09626ce418864ce1b",
        "domain": "omni",
        "question": "Consider the two hands of an analog clock, each of which moves with constant angular velocity. Certain positions of these hands are possible (e.g. the hour hand halfway between the 5 and 6 and the minute hand exactly at the 6), while others are impossible (e.g. the hour hand exactly at the 5 and the minute hand exactly at the 6). How many different positions are there that would remain possible if the hour and minute hands were switched?",
        "answer": "143",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 779,
        "difficulty": 5.0
      },
      {
        "id": "f5dfbf7f282d8dc8e0c684077f579f087b9f95381c6911c5c71a9914894299d9",
        "domain": "omni",
        "question": "A rectangular piece of paper $P Q R S$ has $P Q=20$ and $Q R=15$. The piece of paper is glued flat on the surface of a large cube so that $Q$ and $S$ are at vertices of the cube. What is the shortest distance from $P$ to $R$, as measured through the cube?",
        "answer": "18.4",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "fermat",
        "source_index": 3128,
        "difficulty": 2.5
      },
      {
        "id": "ad42ee9e773ede4096d9c1816b94806bf1adac8904cbb15ca2f1e1096e6d23d8",
        "domain": "omni",
        "question": "How many ways can you mark 8 squares of an $8 \\times 8$ chessboard so that no two marked squares are in the same row or column, and none of the four corner squares is marked? (Rotations and reflections are considered different.)",
        "answer": "21600",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_2",
        "source_index": 1117,
        "difficulty": 4.0
      },
      {
        "id": "e6bd0257de821357a144987e88da5da294ff561a9c179ac3bc0b0bd13a440217",
        "domain": "omni",
        "question": "Suppose $a, b$, and $c$ are real numbers such that $$\\begin{aligned} a^{2}-b c & =14 \\\\ b^{2}-c a & =14, \\text { and } \\\\ c^{2}-a b & =-3 \\end{aligned}$$ Compute $|a+b+c|$.",
        "answer": "\\frac{17}{5}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1788,
        "difficulty": 5.0
      },
      {
        "id": "1a3c0c9664476e4e4b9a84257cafc79bd2c27a88814cbf83c3ad89fbb8c45b9e",
        "domain": "omni",
        "question": "Danielle picks a positive integer $1 \\leq n \\leq 2016$ uniformly at random. What is the probability that \\operatorname{gcd}(n, 2015)=1?",
        "answer": "\\frac{1441}{2016}",
        "topic": "Mathematics -> Number Theory -> Greatest Common Divisors (GCD); Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1867,
        "difficulty": 4.5
      },
      {
        "id": "10fb1ef1d70c6ade64cc9ec3f55f807c0b97bbaa3ab0c1f09fbc9d4667bd8ad2",
        "domain": "omni",
        "question": "Find $AB + AC$ in triangle $ABC$ given that $D$ is the midpoint of $BC$, $E$ is the midpoint of $DC$, and $BD = DE = EA = AD$.",
        "answer": "1+\\frac{\\sqrt{3}}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2377,
        "difficulty": 3.5
      },
      {
        "id": "2588e15aeca1083ceb2d4738d474a3ff55377fbb3d0ee9c198ade9c0e8545885",
        "domain": "omni",
        "question": "There are two red, two black, two white, and a positive but unknown number of blue socks in a drawer. It is empirically determined that if two socks are taken from the drawer without replacement, the probability they are of the same color is $\\frac{1}{5}$. How many blue socks are there in the drawer?",
        "answer": "4",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "HMMT_2",
        "source_index": 957,
        "difficulty": 3.5
      },
      {
        "id": "adefcadd733f4cb3c5ea0f45797842d0f1bc855f86009dfd1c58be19ed33b642",
        "domain": "omni",
        "question": "What is $x-y$ if a town has 2017 houses, 1820 have a dog, 1651 have a cat, 1182 have a turtle, $x$ is the largest possible number of houses that have a dog, a cat, and a turtle, and $y$ is the smallest possible number of houses that have a dog, a cat, and a turtle?",
        "answer": "563",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 2992,
        "difficulty": 2.0
      },
      {
        "id": "6927bddfb87fd2634e7d9c3eaebe73d43bc3f04b977f66c9f0939086eddc4f59",
        "domain": "omni",
        "question": "In how many distinct ways can you color each of the vertices of a tetrahedron either red, blue, or green such that no face has all three vertices the same color? (Two colorings are considered the same if one coloring can be rotated in three dimensions to obtain the other.)",
        "answer": "6",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1487,
        "difficulty": 5.0
      },
      {
        "id": "0286ce8b90c2a430750ee3b270817c0c627c7ef18c4b9e4479638f20e8a48d04",
        "domain": "omni",
        "question": "In how many different ways can André form exactly \\( \\$10 \\) using \\( \\$1 \\) coins, \\( \\$2 \\) coins, and \\( \\$5 \\) bills?",
        "answer": "10",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 2873,
        "difficulty": 1.5
      },
      {
        "id": "662990d2bf6a4b268e609ec21054c4c8d0137066849045ce40b2d107f3646e84",
        "domain": "omni",
        "question": "Azmi has four blocks, each in the shape of a rectangular prism and each with dimensions $2 \times 3 \times 6$. She carefully stacks these four blocks on a flat table to form a tower that is four blocks high. What is the number of possible heights for this tower?",
        "answer": "14",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "pascal",
        "source_index": 2998,
        "difficulty": 2.0
      },
      {
        "id": "cdb9540ce4895f1c64ba39534f7e767e5fd088d5c28cfc79f4262f91e345000b",
        "domain": "omni",
        "question": "Given $\\frac{e}{f}=\\frac{3}{4}$ and $\\sqrt{e^{2}+f^{2}}=15$, find $ef$.",
        "answer": "108",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 931,
        "difficulty": 5.0
      },
      {
        "id": "195e4ec17d5bbba8c941270528fc38f1bec433947a7c52ee4bca3511b62b2967",
        "domain": "omni",
        "question": "Let $M$ denote the number of positive integers which divide 2014!, and let $N$ be the integer closest to $\\ln (M)$. Estimate the value of $N$. If your answer is a positive integer $A$, your score on this problem will be the larger of 0 and $\\left\\lfloor 20-\\frac{1}{8}|A-N|\\right\\rfloor$. Otherwise, your score will be zero.",
        "answer": "439",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_11",
        "source_index": 1743,
        "difficulty": 5.0
      },
      {
        "id": "3014c7bb8fa242fa5a09619b9550fd366b31405400e5cb60fb0d77bfe34ab71b",
        "domain": "omni",
        "question": "If $m, n$ and $p$ are positive integers with $m+\\frac{1}{n+\\frac{1}{p}}=\\frac{17}{3}$, what is the value of $n$?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 3209,
        "difficulty": 2.5
      },
      {
        "id": "04b513b57bb32f621ad04781931b9cc389a2ae1b9f59d01f8aa96f43893ccccf",
        "domain": "omni",
        "question": "Find all positive integers $n$ for which all positive divisors of $n$ can be put into the cells of a rectangular table under the following constraints:\n[list]\n[*]each cell contains a distinct divisor;\n[*]the sums of all rows are equal; and\n[*]the sums of all columns are equal.\n[/list]",
        "answer": "1",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "imo_shortlist",
        "source_index": 4021,
        "difficulty": 9.0
      },
      {
        "id": "a2863e804b792a4a51b6aa894ebca212bb5eb9a58c5b6f5c5b029e5d318b1fce",
        "domain": "omni",
        "question": "Find the number of ordered triples of divisors $(d_{1}, d_{2}, d_{3})$ of 360 such that $d_{1} d_{2} d_{3}$ is also a divisor of 360.",
        "answer": "800",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_11",
        "source_index": 1982,
        "difficulty": 4.5
      },
      {
        "id": "8d9e445381e6baf77f80acc951919471ed0c6d6e383f49ecedcf2bcfee93c7c9",
        "domain": "omni",
        "question": "Compute $\\frac{x}{w}$ if $w \\neq 0$ and $\\frac{x+6 y-3 z}{-3 x+4 w}=\\frac{-2 y+z}{x-w}=\\frac{2}{3}$.",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_11",
        "source_index": 2631,
        "difficulty": 5.0
      },
      {
        "id": "1de48c2554ff0e1100f856cdac1e7976122a3132f58582aa6d41f691d3723573",
        "domain": "omni",
        "question": "A positive integer $n$ is picante if $n$ ! ends in the same number of zeroes whether written in base 7 or in base 8 . How many of the numbers $1,2, \\ldots, 2004$ are picante?",
        "answer": "4",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 854,
        "difficulty": 5.25
      },
      {
        "id": "e7889acc454fd4b0a48a3efea1d1cdc35d1a70ad4eb9f7b36333fb18a1b8e700",
        "domain": "omni",
        "question": "Find the number of ordered triples of positive integers $(a, b, c)$ such that $6a+10b+15c=3000$.",
        "answer": "4851",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 442,
        "difficulty": 5.25
      },
      {
        "id": "2db79618e8ba76b7377e244bdaf09db273335fc38cef3a869c19814e46278cbd",
        "domain": "omni",
        "question": "Let $A B C D$ be an isosceles trapezoid with $A D=B C=255$ and $A B=128$. Let $M$ be the midpoint of $C D$ and let $N$ be the foot of the perpendicular from $A$ to $C D$. If $\\angle M B C=90^{\\circ}$, compute $\\tan \\angle N B M$.",
        "answer": "\\frac{120}{353}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2177,
        "difficulty": 4.0
      },
      {
        "id": "2b283ea9b7a724b8e5efccd13e2f82d3add0d8f8bd9cd40b36622825dd4897f7",
        "domain": "omni",
        "question": "In a rectangle $P Q R S$ with $P Q=5$ and $Q R=3$, $P R$ is divided into three segments of equal length by points $T$ and $U$. What is the area of quadrilateral $S T Q U$?",
        "answer": "5",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "fermat",
        "source_index": 3163,
        "difficulty": 2.5
      },
      {
        "id": "61fed47d2fc54687fa592e5017af04bccb8028b4cd26f8b90b4a5e02b9ca5639",
        "domain": "omni",
        "question": "Let $A$ denote the set of all integers $n$ such that $1 \\leq n \\leq 10000$, and moreover the sum of the decimal digits of $n$ is 2. Find the sum of the squares of the elements of $A$.",
        "answer": "7294927",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "HMMT_2",
        "source_index": 1312,
        "difficulty": 4.0
      },
      {
        "id": "83f2fdc414ac94986f656c2362aaa63f749f56bfdabbfe949f9a4e84355c52b6",
        "domain": "omni",
        "question": "In the country of Francisca, there are 2010 cities, some of which are connected by roads. Between any two cities, there is a unique path which runs along the roads and which does not pass through any city twice. What is the maximum possible number of cities in Francisca which have at least 3 roads running out of them?",
        "answer": "1004",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_2",
        "source_index": 2355,
        "difficulty": 5.25
      },
      {
        "id": "713e5f14a0f2fbcfd35ebedc384525246096974245f42675ea33c3b7771d546c",
        "domain": "omni",
        "question": "One million bucks (i.e. one million male deer) are in different cells of a $1000 \\times 1000$ grid. The left and right edges of the grid are then glued together, and the top and bottom edges of the grid are glued together, so that the grid forms a doughnut-shaped torus. Furthermore, some of the bucks are honest bucks, who always tell the truth, and the remaining bucks are dishonest bucks, who never tell the truth. Each of the million bucks claims that \"at most one of my neighboring bucks is an honest buck.\" A pair of neighboring bucks is said to be buckaroo if exactly one of them is an honest buck. What is the minimum possible number of buckaroo pairs in the grid?",
        "answer": "1200000",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2231,
        "difficulty": 4.5
      },
      {
        "id": "f2ef05ff7d0ee3ef8d6a64f4af16405ac7589af26229daa50a252aec9c62a2c3",
        "domain": "omni",
        "question": "For how many pairs of nonzero integers $(c, d)$ with $-2015 \\leq c, d \\leq 2015$ do the equations $c x=d$ and $d x=c$ both have an integer solution?",
        "answer": "8060",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 1745,
        "difficulty": 5.0
      },
      {
        "id": "093f15ef94cde3d56708fd092a20f2678257d8a1ee5edcac2c7c5926cc81cad9",
        "domain": "omni",
        "question": "Find the largest integer less than 2012 all of whose divisors have at most two 1's in their binary representations.",
        "answer": "1536",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_11",
        "source_index": 1984,
        "difficulty": 4.5
      },
      {
        "id": "3823d17646ce86ffdc75da68b3b4815ca44e9fb5a9e072aef1ba791fce49656b",
        "domain": "omni",
        "question": "Let $P$ and $Q$ be points on line $l$ with $P Q=12$. Two circles, $\\omega$ and $\\Omega$, are both tangent to $l$ at $P$ and are externally tangent to each other. A line through $Q$ intersects $\\omega$ at $A$ and $B$, with $A$ closer to $Q$ than $B$, such that $A B=10$. Similarly, another line through $Q$ intersects $\\Omega$ at $C$ and $D$, with $C$ closer to $Q$ than $D$, such that $C D=7$. Find the ratio $A D / B C$.",
        "answer": "\\frac{8}{9}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 2546,
        "difficulty": 5.0
      },
      {
        "id": "4d4bc73cefd1c1a118af1a127c72e203fc7d1d66f6b5457a4c5360c5bcb92493",
        "domain": "omni",
        "question": "Roger initially has 20 socks in a drawer, each of which is either white or black. He chooses a sock uniformly at random from the drawer and throws it away. He repeats this action until there are equal numbers of white and black socks remaining. Suppose that the probability he stops before all socks are gone is $p$. If the sum of all distinct possible values of $p$ over all initial combinations of socks is $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$",
        "answer": "20738",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 869,
        "difficulty": 5.25
      },
      {
        "id": "aafface7a737dc36b1031f493ca4af283f84c97baa30fb614bc063efaf9189cc",
        "domain": "omni",
        "question": "A positive integer $n$ is magical if $\\lfloor\\sqrt{\\lceil\\sqrt{n}\\rceil}\\rfloor=\\lceil\\sqrt{\\lfloor\\sqrt{n}\\rfloor}\\rceil$ where $\\lfloor\\cdot\\rfloor$ and $\\lceil\\cdot\\rceil$ represent the floor and ceiling function respectively. Find the number of magical integers between 1 and 10,000, inclusive.",
        "answer": "1330",
        "topic": "Mathematics -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 2106,
        "difficulty": 4.0
      },
      {
        "id": "96555e0755a00322ca6d3dea0de84243f84aa844a83de2915a1ea811f04cd069",
        "domain": "omni",
        "question": "There are $10$ birds on the ground. For any $5$ of them, there are at least $4$ birds on a circle. Determine the least possible number of birds on the circle with the most birds.",
        "answer": "9",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "china_national_olympiad",
        "source_index": 63,
        "difficulty": 6.0
      },
      {
        "id": "4f6df5a64e6b856f0079862d38f45e5898d58dca85f48ddd2927cc29b2e8a3f0",
        "domain": "omni",
        "question": "What is the tens digit of the smallest six-digit positive integer that is divisible by each of $10,11,12,13,14$, and 15?",
        "answer": "2",
        "topic": "Mathematics -> Number Theory -> Least Common Multiples (LCM)",
        "source": "pascal",
        "source_index": 2924,
        "difficulty": 2.0
      },
      {
        "id": "85ad61ba445988d71e41cf13ccaf84c49a4bfd5dcd5c8118e6e9feee03436c1c",
        "domain": "omni",
        "question": "Eugene swam on Sunday, Monday, and Tuesday. On Monday, he swam for 30 minutes. On Tuesday, he swam for 45 minutes. His average swim time over the three days was 34 minutes. For how many minutes did he swim on Sunday?",
        "answer": "27",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "cayley",
        "source_index": 3119,
        "difficulty": 1.5
      },
      {
        "id": "1a38e1ee3c57b2585d2117085592074b756d1ed9bf70674510afb39a4ea76610",
        "domain": "omni",
        "question": "A numerical value is assigned to each letter of the alphabet. The value of a word is determined by adding up the numerical values of each of its letters. The value of SET is 2, the value of HAT is 7, the value of TASTE is 3, and the value of MAT is 4. What is the value of the word MATH?",
        "answer": "10",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3071,
        "difficulty": 1.0
      },
      {
        "id": "aff46d0e0defaa05461c35b5990f3c64ddb4959e4ac34178069c9f68f4a116be",
        "domain": "omni",
        "question": "Jarris is a weighted tetrahedral die with faces $F_{1}, F_{2}, F_{3}, F_{4}$. He tosses himself onto a table, so that the probability he lands on a given face is proportional to the area of that face. Let $k$ be the maximum distance any part of Jarris is from the table after he rolls himself. Given that Jarris has an inscribed sphere of radius 3 and circumscribed sphere of radius 10, find the minimum possible value of the expected value of $k$.",
        "answer": "12",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 810,
        "difficulty": 4.75
      },
      {
        "id": "d084192afcaad6dadfe26ca1bf6b85122209113ca4136906f9422d5f71f28638",
        "domain": "omni",
        "question": "What is the number of positive integers $p$ for which $-1<\\sqrt{p}-\\sqrt{100}<1$?",
        "answer": "39",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Inequalities",
        "source": "fermat",
        "source_index": 3136,
        "difficulty": 2.5
      },
      {
        "id": "ad8c3ad1fdc47c30966612901d5d3e7aa57202cbe42ef5ba3aa90594d106b5be",
        "domain": "omni",
        "question": "A point is equidistant from the coordinate axes if the vertical distance from the point to the $x$-axis is equal to the horizontal distance from the point to the $y$-axis. The point of intersection of the vertical line $x = a$ with the line with equation $3x + 8y = 24$ is equidistant from the coordinate axes. What is the sum of all possible values of $a$?",
        "answer": "-\\frac{144}{55}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Lines -> Other; Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "fermat",
        "source_index": 2737,
        "difficulty": 2.5
      },
      {
        "id": "4a28a7595b862ba7bbae039197968b4742f7e093093f8e2042f3136bf8687dc4",
        "domain": "omni",
        "question": "Find the sum of all real numbers $x$ such that $5 x^{4}-10 x^{3}+10 x^{2}-5 x-11=0$.",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "HMMT_2",
        "source_index": 1260,
        "difficulty": 5.0
      },
      {
        "id": "46d9013d16cb4762115991d4bfe9e6228b0b09b3ce81c28c13b310ced3220ad7",
        "domain": "omni",
        "question": "Let a positive integer \\(n\\) be called a cubic square if there exist positive integers \\(a, b\\) with \\(n=\\operatorname{gcd}\\left(a^{2}, b^{3}\\right)\\). Count the number of cubic squares between 1 and 100 inclusive.",
        "answer": "13",
        "topic": "Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 3369,
        "difficulty": 5.0
      },
      {
        "id": "c33160bf2ce0221c22c56c733ce4f414b7d513f8e4619300f392a7cd37f4c25d",
        "domain": "omni",
        "question": "The very hungry caterpillar lives on the number line. For each non-zero integer $i$, a fruit sits on the point with coordinate $i$. The caterpillar moves back and forth; whenever he reaches a point with food, he eats the food, increasing his weight by one pound, and turns around. The caterpillar moves at a speed of $2^{-w}$ units per day, where $w$ is his weight. If the caterpillar starts off at the origin, weighing zero pounds, and initially moves in the positive $x$ direction, after how many days will he weigh 10 pounds?",
        "answer": "9217",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems; Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "HMMT_11",
        "source_index": 1860,
        "difficulty": 5.0
      },
      {
        "id": "24fd5676bab02597bb12c78ae3cb10a2cc0fcdf3de9e893d070b60d967ecf130",
        "domain": "omni",
        "question": "Let $Z$ be as in problem 15. Let $X$ be the greatest integer such that $|X Z| \\leq 5$. Find $X$.",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 877,
        "difficulty": 5.0
      },
      {
        "id": "223854ecc53bcb7eeee24263434279490f1ba8bdb63b3f9a76bd9f069eb61b5f",
        "domain": "omni",
        "question": "The cells of a $5 \\times 5$ grid are each colored red, white, or blue. Sam starts at the bottom-left cell of the grid and walks to the top-right cell by taking steps one cell either up or to the right. Thus, he passes through 9 cells on his path, including the start and end cells. Compute the number of colorings for which Sam is guaranteed to pass through a total of exactly 3 red cells, exactly 3 white cells, and exactly 3 blue cells no matter which route he takes.",
        "answer": "1680",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1719,
        "difficulty": 5.0
      },
      {
        "id": "c0acca689b33da83366316b20da5e699aa7bca1953d24800954905601c10073d",
        "domain": "omni",
        "question": "Compute the number of dates in the year 2023 such that when put in MM/DD/YY form, the three numbers are in strictly increasing order. For example, $06 / 18 / 23$ is such a date since $6<18<23$, while today, $11 / 11 / 23$, is not.",
        "answer": "186",
        "topic": "Mathematics -> Applied Mathematics -> Math Word Problems",
        "source": "HMMT_11",
        "source_index": 1878,
        "difficulty": 3.5
      },
      {
        "id": "7e4ad5d19c74cbcf203ec2335ea110f17b1530afcc246f7887d300f023d586e9",
        "domain": "omni",
        "question": "Altitudes $B E$ and $C F$ of acute triangle $A B C$ intersect at $H$. Suppose that the altitudes of triangle $E H F$ concur on line $B C$. If $A B=3$ and $A C=4$, then $B C^{2}=\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
        "answer": "33725",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations; Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_11",
        "source_index": 2578,
        "difficulty": 4.5
      },
      {
        "id": "b5ea7cbf590ca065e51a03816d016bafe98bb296cadcc4d694886b68db3ab068",
        "domain": "omni",
        "question": "What is the probability that the arrow stops on a shaded region if a circular spinner is divided into six regions, four regions each have a central angle of $x^{\\circ}$, and the remaining regions have central angles of $20^{\\circ}$ and $140^{\\circ}$?",
        "answer": "\\frac{2}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "pascal",
        "source_index": 2972,
        "difficulty": 2.0
      },
      {
        "id": "4dadfec2ab98897ba06ffe57e646cd6f25495d10211fc00257f87e930ef65261",
        "domain": "omni",
        "question": "Compute the number of positive integers $n \\leq 1000$ such that \\operatorname{lcm}(n, 9)$ is a perfect square.",
        "answer": "43",
        "topic": "Mathematics -> Number Theory -> Least Common Multiples (LCM)",
        "source": "HMMT_2",
        "source_index": 1216,
        "difficulty": 4.0
      },
      {
        "id": "3bb6bf56eb5e0f2632ae77fca1f22828d361c438ab21f1750add1a168f91eac2",
        "domain": "omni",
        "question": "In the Cartesian plane, a perfectly reflective semicircular room is bounded by the upper half of the unit circle centered at $(0,0)$ and the line segment from $(-1,0)$ to $(1,0)$. David stands at the point $(-1,0)$ and shines a flashlight into the room at an angle of $46^{\\circ}$ above the horizontal. How many times does the light beam reflect off the walls before coming back to David at $(-1,0)$ for the first time?",
        "answer": "65",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2220,
        "difficulty": 4.0
      },
      {
        "id": "59480ef65c3388083d3ce0f0a8bbbb0da73d9f557e063db194ff968627dbf79c",
        "domain": "omni",
        "question": "Let $f(x)=x^{3}-3x$. Compute the number of positive divisors of $$\\left\\lfloor f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(\\frac{5}{2}\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\rfloor$$ where $f$ is applied 8 times.",
        "answer": "6562",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Exponential Functions",
        "source": "HMMT_2",
        "source_index": 888,
        "difficulty": 5.25
      },
      {
        "id": "886e576ede7a4b4b78824bc5fd6b02e7496033d003f95385fd1d4434a7bfb1fa",
        "domain": "omni",
        "question": "There is a heads up coin on every integer of the number line. Lucky is initially standing on the zero point of the number line facing in the positive direction. Lucky performs the following procedure: he looks at the coin (or lack thereof) underneath him, and then, - If the coin is heads up, Lucky flips it to tails up, turns around, and steps forward a distance of one unit. - If the coin is tails up, Lucky picks up the coin and steps forward a distance of one unit facing the same direction. - If there is no coin, Lucky places a coin heads up underneath him and steps forward a distance of one unit facing the same direction. He repeats this procedure until there are 20 coins anywhere that are tails up. How many times has Lucky performed the procedure when the process stops?",
        "answer": "6098",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1174,
        "difficulty": 6.5
      },
      {
        "id": "c25feba593b31b2188a4f009451d8c9877076dab170e470585f55de5367dc228",
        "domain": "omni",
        "question": "Find the smallest constant $C$ such that for every real polynomial $P(x)$ of degree 3 that has a root in the interval $[0,1]$, \\[ \\int_0^1 \\left| P(x) \\right|\\,dx \\leq C \\max_{x \\in [0,1]} \\left| P(x) \\right|. \\]",
        "answer": "\\frac{5}{6}",
        "topic": "Mathematics -> Calculus -> Integral Calculus -> Applications of Integrals; Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "putnam",
        "source_index": 3233,
        "difficulty": 8.0
      },
      {
        "id": "f8c8cef5bf43010c5451e407b2e96a76451f3a39ff245c6b8cf3e6c1d327e66e",
        "domain": "omni",
        "question": "A sequence consists of 2010 terms. Each term after the first is 1 larger than the previous term. The sum of the 2010 terms is 5307. What is the sum when every second term is added up, starting with the first term and ending with the second last term?",
        "answer": "2151",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Sequences -> Other",
        "source": "cayley",
        "source_index": 3029,
        "difficulty": 2.0
      },
      {
        "id": "b72a01d3bf01e8432c429f0594b5978f4df56bcaf79f438893dac0f42e74897e",
        "domain": "omni",
        "question": "Find the 6-digit number beginning and ending in the digit 2 that is the product of three consecutive even integers.",
        "answer": "287232",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 970,
        "difficulty": 4.5
      },
      {
        "id": "6acdf2907bdef46c6da89e8ea39b57bfb5d487b3ae8cc22bdbc039aabbd840fb",
        "domain": "omni",
        "question": "For a positive integer $n$, let $p(n)$ denote the product of the positive integer factors of $n$. Determine the number of factors $n$ of 2310 for which $p(n)$ is a perfect square.",
        "answer": "27",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_11",
        "source_index": 2055,
        "difficulty": 4.0
      },
      {
        "id": "e0bdb351d5e82693084211789b97f497fbd2636b216dbcbb99add0d74cc4ea77",
        "domain": "omni",
        "question": "Let $S$ be a subset with four elements chosen from \\{1,2, \\ldots, 10\\}$. Michael notes that there is a way to label the vertices of a square with elements from $S$ such that no two vertices have the same label, and the labels adjacent to any side of the square differ by at least 4 . How many possibilities are there for the subset $S$ ?",
        "answer": "36",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2076,
        "difficulty": 4.0
      },
      {
        "id": "23aa557dec1529a516f8c18d97a4a401f04d63bf91d59b38cc33002c43878d69",
        "domain": "omni",
        "question": "For what single digit $n$ does 91 divide the 9-digit number $12345 n 789$?",
        "answer": "7",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Other",
        "source": "HMMT_2",
        "source_index": 3351,
        "difficulty": 4.5
      },
      {
        "id": "cfb37f3ee13b8146641e65c843a871727d9a2ee5618303084398d36b53fa1b86",
        "domain": "omni",
        "question": "For each positive integer $1 \\leq m \\leq 10$, Krit chooses an integer $0 \\leq a_{m}<m$ uniformly at random. Let $p$ be the probability that there exists an integer $n$ for which $n \\equiv a_{m}(\\bmod m)$ for all $m$. If $p$ can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
        "answer": "1540",
        "topic": "Mathematics -> Number Theory -> Congruences",
        "source": "HMMT_2",
        "source_index": 850,
        "difficulty": 5.25
      },
      {
        "id": "4014c207f3ef670b15fe0849db6383e5db79528d985b886d90151c2335c2d601",
        "domain": "omni",
        "question": "For how many positive integers $a$ does the polynomial $x^{2}-a x+a$ have an integer root?",
        "answer": "1",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Quadratic Functions",
        "source": "HMMT_11",
        "source_index": 2102,
        "difficulty": 5.0
      },
      {
        "id": "f52e2b1f87f91b92c2badb5048721b56ff111c0d45d5cf1f77464353f350ce6c",
        "domain": "omni",
        "question": "Reimu has a wooden cube. In each step, she creates a new polyhedron from the previous one by cutting off a pyramid from each vertex of the polyhedron along a plane through the trisection point on each adjacent edge that is closer to the vertex. For example, the polyhedron after the first step has six octagonal faces and eight equilateral triangular faces. How many faces are on the polyhedron after the fifth step?",
        "answer": "974",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_11",
        "source_index": 2104,
        "difficulty": 4.0
      },
      {
        "id": "8cf53644a12b8f799e118f26f00c1365b780c1234a80e0f1e361a83b0d1a90a6",
        "domain": "omni",
        "question": "Let $P$ be the number to partition 2013 into an ordered tuple of prime numbers? What is $\\log _{2}(P)$? If your answer is $A$ and the correct answer is $C$, then your score on this problem will be $\\left\\lfloor\\frac{125}{2}\\left(\\min \\left(\\frac{C}{A}, \\frac{A}{C}\\right)-\\frac{3}{5}\\right)\\right\\rfloor$ or zero, whichever is larger.",
        "answer": "614.519...",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Prealgebra -> Other",
        "source": "HMMT_2",
        "source_index": 614,
        "difficulty": 5.0
      },
      {
        "id": "ee76b9a207dbb967236f961545cfa10baa712c87451faf698c851c3b64e7f8f4",
        "domain": "omni",
        "question": "Let $A B C$ be a triangle and $D, E$, and $F$ be the midpoints of sides $B C, C A$, and $A B$ respectively. What is the maximum number of circles which pass through at least 3 of these 6 points?",
        "answer": "17",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Circles; Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 517,
        "difficulty": 5.0
      },
      {
        "id": "25a1d0446077c0649099fe6c26a1ead6d0c1c573d02254c36a0455068993d396",
        "domain": "omni",
        "question": "Let $A$ be as in problem 33. Let $W$ be the sum of all positive integers that divide $A$. Find $W$.",
        "answer": "8",
        "topic": "Mathematics -> Number Theory -> Other (since the context of \\( A \\) is necessary but unspecified here, the question relates to determining and summing all divisors of an integer) -> Other",
        "source": "HMMT_2",
        "source_index": 610,
        "difficulty": 4.5
      },
      {
        "id": "dca3ac4974026a2c76dc5560ed67763eb1bfd8c44ed5fa198463670823879d1e",
        "domain": "omni",
        "question": "In quadrilateral $ABCD$, there exists a point $E$ on segment $AD$ such that $\\frac{AE}{ED}=\\frac{1}{9}$ and $\\angle BEC$ is a right angle. Additionally, the area of triangle $CED$ is 27 times more than the area of triangle $AEB$. If $\\angle EBC=\\angle EAB, \\angle ECB=\\angle EDC$, and $BC=6$, compute the value of $AD^{2}$.",
        "answer": "320",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Angles; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2625,
        "difficulty": 5.25
      },
      {
        "id": "4ad66868c710a6406fc6b1050cf6f69f4f41497a5506687c2b70f5605b0167b9",
        "domain": "omni",
        "question": "Compute the side length of the largest cube contained in the region $\\{(x, y, z): x^{2}+y^{2}+z^{2} \\leq 25 \\text{ and } x \\geq 0\\}$ of three-dimensional space.",
        "answer": "\\frac{5 \\sqrt{6}}{3}",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 1108,
        "difficulty": 4.0
      },
      {
        "id": "1c6b69da9a542af77216a9a2a5187b146ae2e638705baa18f05fe78ce09da229",
        "domain": "omni",
        "question": "Ten points are equally spaced on a circle. A graph is a set of segments (possibly empty) drawn between pairs of points, so that every two points are joined by either zero or one segments. Two graphs are considered the same if we can obtain one from the other by rearranging the points. Let $N$ denote the number of graphs with the property that for any two points, there exists a path from one to the other among the segments of the graph. Estimate the value of $N$. If your answer is a positive integer $A$, your score on this problem will be the larger of 0 and $\\lfloor 20-5|\\ln (A / N)|\\rfloor$. Otherwise, your score will be zero.",
        "answer": "11716571",
        "topic": "Mathematics -> Discrete Mathematics -> Graph Theory",
        "source": "HMMT_11",
        "source_index": 1795,
        "difficulty": 6.0
      },
      {
        "id": "86189385a26c9245f25191bf27825dc461b38f8242fd0a8c616b9051e7ddbce4",
        "domain": "omni",
        "question": "Mary has a sequence $m_{2}, m_{3}, m_{4}, \\ldots$, such that for each $b \\geq 2, m_{b}$ is the least positive integer $m$ for which none of the base-$b$ logarithms $\\log _{b}(m), \\log _{b}(m+1), \\ldots, \\log _{b}(m+2017)$ are integers. Find the largest number in her sequence.",
        "answer": "2188",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Logarithmic Functions",
        "source": "HMMT_11",
        "source_index": 2430,
        "difficulty": 4.0
      },
      {
        "id": "b396f8e00594e3f637b40ffd3c3474752bd2d9cef09f0b409370fa2040b21cdd",
        "domain": "omni",
        "question": "Let $ \\left(a_{n}\\right)$ be the sequence of reals defined by $ a_{1}=\\frac{1}{4}$ and the recurrence $ a_{n}= \\frac{1}{4}(1+a_{n-1})^{2}, n\\geq 2$. Find the minimum real $ \\lambda$ such that for any non-negative reals $ x_{1},x_{2},\\dots,x_{2002}$, it holds\r\n\\[ \\sum_{k=1}^{2002}A_{k}\\leq \\lambda a_{2002}, \\]\r\nwhere $  A_{k}= \\frac{x_{k}-k}{(x_{k}+\\cdots+x_{2002}+\\frac{k(k-1)}{2}+1)^{2}}, k\\geq 1$.",
        "answer": "\\frac{1}{2005004}",
        "topic": "Mathematics -> Algebra -> Intermediate Algebra -> Recursive Sequences -> Other",
        "source": "china_team_selection_test",
        "source_index": 261,
        "difficulty": 8.0
      },
      {
        "id": "080453a1206d271f1d6e7b40eb56ce5669210a0be2775b6ae142b1849cec95d5",
        "domain": "omni",
        "question": "Abigail chooses an integer at random from the set $\\{2,4,6,8,10\\}$. Bill chooses an integer at random from the set $\\{2,4,6,8,10\\}$. Charlie chooses an integer at random from the set $\\{2,4,6,8,10\\}$. What is the probability that the product of their three integers is not a power of 2?",
        "answer": "\\frac{98}{125}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "cayley",
        "source_index": 2671,
        "difficulty": 2.0
      },
      {
        "id": "53d8a7c74fe88867529c1b44eea3543e485f6350cfb6b770c9574e3bc1c336bc",
        "domain": "omni",
        "question": "For how many values of $n$ with $3 \\leq n \\leq 12$ can a Fano table be created?",
        "answer": "3",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Other",
        "source": "cayley",
        "source_index": 3193,
        "difficulty": 2.5
      },
      {
        "id": "93291c48a3e6f08624a05404dab8d4e96f3bfda1ef0d1ac5f6a48e2a1a675c77",
        "domain": "omni",
        "question": "Distinct prime numbers $p, q, r$ satisfy the equation $2 p q r+50 p q=7 p q r+55 p r=8 p q r+12 q r=A$ for some positive integer $A$. What is $A$ ?",
        "answer": "1980",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1687,
        "difficulty": 4.5
      },
      {
        "id": "a34a54356ed4509215c7854be0f02a8f1ca58a3942ffe61c7e16e92af487a6d7",
        "domain": "omni",
        "question": "Let $a$ and $b$ be positive real numbers such that $3a^2 + 2b^2 = 3a + 2b$. Find the minimum value of $A =\\sqrt{\\frac{a}{b(3a+2)}} + \\sqrt{\\frac{b}{a(2b+3)}} $",
        "answer": "\\frac{2}{\\sqrt{5}}",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities; Mathematics -> Calculus -> Differential Calculus -> Applications of Derivatives",
        "source": "jbmo_shortlist",
        "source_index": 3949,
        "difficulty": 6.0
      },
      {
        "id": "42a0b3b34cf12ebdb880f0cff77f67ba2ad67c2870ca1c08b6ba4c6162bfc09e",
        "domain": "omni",
        "question": "Given that the 32-digit integer 64312311692944269609355712372657 is the product of 6 consecutive primes, compute the sum of these 6 primes.",
        "answer": "1200974",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 709,
        "difficulty": 5.0
      },
      {
        "id": "121ed78bd1ba2e8d430c8161c5f8a7f1c3978f7fee1195313caa614621df969c",
        "domain": "omni",
        "question": "The Dyslexian alphabet consists of consonants and vowels. It so happens that a finite sequence of letters is a word in Dyslexian precisely if it alternates between consonants and vowels (it may begin with either). There are 4800 five-letter words in Dyslexian. How many letters are in the alphabet?",
        "answer": "12",
        "topic": "Mathematics -> Algebra -> Algebra -> Equations and Inequalities",
        "source": "HMMT_2",
        "source_index": 789,
        "difficulty": 3.5
      },
      {
        "id": "c52b79146142bce1f7de6e8417894d510a66679966438ca74c016ae9e09a1f6f",
        "domain": "omni",
        "question": "How many functions $f:\\{1,2, \\ldots, 10\\} \\rightarrow\\{1,2, \\ldots, 10\\}$ satisfy the property that $f(i)+f(j)=11$ for all values of $i$ and $j$ such that $i+j=11$.",
        "answer": "100000",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_11",
        "source_index": 2510,
        "difficulty": 5.25
      },
      {
        "id": "065b1dabf3a120401e70340dedc3285a445730b10f157f83c387e4e676327516",
        "domain": "omni",
        "question": "We consider positive integers $n$ having at least six positive divisors. Let the positive divisors of $n$ be arranged in a sequence $(d_i)_{1\\le i\\le k}$ with $$1=d_1<d_2<\\dots <d_k=n\\quad (k\\ge 6).$$\nFind all positive integers $n$ such that $$n=d_5^2+d_6^2.$$",
        "answer": "500",
        "topic": "Mathematics -> Number Theory -> Divisors -> Other",
        "source": "czech-polish-slovak matches",
        "source_index": 3683,
        "difficulty": 7.0
      },
      {
        "id": "238591be780a908de5b632c49cd481b8ef44413e149fcda95039e856d0f86481",
        "domain": "omni",
        "question": "A lock code is made up of four digits that satisfy the following rules: - At least one digit is a 4, but neither the second digit nor the fourth digit is a 4. - Exactly one digit is a 2, but the first digit is not 2. - Exactly one digit is a 7. - The code includes a 1, or the code includes a 6, or the code includes two 4s. How many codes are possible?",
        "answer": "22",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations",
        "source": "pascal",
        "source_index": 3067,
        "difficulty": 2.0
      },
      {
        "id": "faa5a0390183fb359711e7eb6d43dd6952a34c7347034f60ba7ead3ed8a7e840",
        "domain": "omni",
        "question": "Our third and final item comes to us from Germany, I mean Geometry. It is known that a regular $n$-gon can be constructed with straightedge and compass if $n$ is a prime that is 1 plus a power of 2. It is also possible to construct a $2 n$-gon whenever an $n$-gon is constructible, or a $p_{1} p_{2} \\cdots p_{m}$-gon where the $p_{i}$ 's are distinct primes of the above form. What is really interesting is that these conditions, together with the fact that we can construct a square, is that they give us all constructible regular $n$-gons. What is the largest $n$ less than $4,300,000,000$ such that a regular $n$-gon is constructible?",
        "answer": "2^{32}",
        "topic": "Mathematics -> Number Theory -> Prime Numbers; Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "HMMT_2",
        "source_index": 1066,
        "difficulty": 7.0
      },
      {
        "id": "93f8792f8e03075dcc6b06e7bcee59f09879e9bd71d1b2ad07b8c80e1c2c591a",
        "domain": "omni",
        "question": "Cyclic pentagon $ABCDE$ has a right angle $\\angle ABC=90^{\\circ}$ and side lengths $AB=15$ and $BC=20$. Supposing that $AB=DE=EA$, find $CD$.",
        "answer": "7",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 540,
        "difficulty": 5.0
      },
      {
        "id": "0961bd8233d4d540361093972b78a958bac4f8a4429f404ab6972f4e9b186915",
        "domain": "omni",
        "question": "Let $P(n)=\\left(n-1^{3}\\right)\\left(n-2^{3}\\right) \\ldots\\left(n-40^{3}\\right)$ for positive integers $n$. Suppose that $d$ is the largest positive integer that divides $P(n)$ for every integer $n>2023$. If $d$ is a product of $m$ (not necessarily distinct) prime numbers, compute $m$.",
        "answer": "48",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1639,
        "difficulty": 6.0
      },
      {
        "id": "0c7c65c591d87256567850f1f30ae77125a327de4a3e2455c5521081c9390364",
        "domain": "omni",
        "question": "Triangle $A B C$ has $A B=4, B C=5$, and $C A=6$. Points $A^{\\prime}, B^{\\prime}, C^{\\prime}$ are such that $B^{\\prime} C^{\\prime}$ is tangent to the circumcircle of $\\triangle A B C$ at $A, C^{\\prime} A^{\\prime}$ is tangent to the circumcircle at $B$, and $A^{\\prime} B^{\\prime}$ is tangent to the circumcircle at $C$. Find the length $B^{\\prime} C^{\\prime}$.",
        "answer": "\\frac{80}{3}",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1997,
        "difficulty": 4.5
      },
      {
        "id": "89687ef1a91680b775e09cabac4394a0e2190db77f6cd2c438a28364a96e21d9",
        "domain": "omni",
        "question": "Mark and William are playing a game. Two walls are placed 1 meter apart, with Mark and William each starting an orb at one of the walls. Simultaneously, they release their orbs directly toward the other. Both orbs are enchanted such that, upon colliding with each other, they instantly reverse direction and go at double their previous speed. Furthermore, Mark has enchanted his orb so that when it collides with a wall it instantly reverses direction and goes at double its previous speed (William's reverses direction at the same speed). Initially, Mark's orb is moving at \\frac{1}{1000} meters/s, and William's orb is moving at 1 meter/s. Mark wins when his orb passes the halfway point between the two walls. How fast, in meters/s, is his orb going when this first happens?",
        "answer": "\\frac{2^{17}}{125}",
        "topic": "Mathematics -> Applied Mathematics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 3320,
        "difficulty": 5.25
      },
      {
        "id": "f57d14d3ba467b6a0e165ad1ea65c8669a175d7c303af9b449a250926139c446",
        "domain": "omni",
        "question": "Let P_{1}, P_{2}, \\ldots, P_{6} be points in the complex plane, which are also roots of the equation x^{6}+6 x^{3}-216=0. Given that P_{1} P_{2} P_{3} P_{4} P_{5} P_{6} is a convex hexagon, determine the area of this hexagon.",
        "answer": "9 \\sqrt{3}",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_11",
        "source_index": 2140,
        "difficulty": 4.0
      },
      {
        "id": "27e130c361ad27199a61e6fbaac8e828bd5855df9ee957411831fae18811464c",
        "domain": "omni",
        "question": "Numbers $m$ and $n$ are on the number line. What is the value of $n-m$?",
        "answer": "55",
        "topic": "Mathematics -> Precalculus -> Functions",
        "source": "pascal",
        "source_index": 2875,
        "difficulty": 1.5
      },
      {
        "id": "dca975774e8956d17ed2e0413baef986934f0c10430fa8985d12934317a221b6",
        "domain": "omni",
        "question": "Find all prime numbers $p$ such that there exists a unique $a \\in \\mathbb{Z}_p$ for which $a^3 - 3a + 1 = 0.$",
        "answer": "3",
        "topic": "Mathematics -> Algebra -> Abstract Algebra -> Group Theory",
        "source": "imc",
        "source_index": 3796,
        "difficulty": 6.0
      },
      {
        "id": "706ab17a5a376b6cfc792c966bb6193c050b04edaa88034d938c341a83978c28",
        "domain": "omni",
        "question": "If $a$ and $b$ are positive integers, the operation $\nabla$ is defined by $a \nabla b=a^{b} \times b^{a}$. What is the value of $2 \nabla 3$?",
        "answer": "72",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Integers",
        "source": "pascal",
        "source_index": 2977,
        "difficulty": 2.0
      },
      {
        "id": "ed2f6d1908c3080dc82971dd22be44d79b5c2674cb2cc5293b704adcb25fe244",
        "domain": "omni",
        "question": "If $a$ , $b$ are integers and $s=a^3+b^3-60ab(a+b)\\geq 2012$ , find the least possible value of $s$.",
        "answer": "2015",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations",
        "source": "jbmo_shortlists",
        "source_index": 3809,
        "difficulty": 5.0
      },
      {
        "id": "b51daa9dfec34345cd34a0e7ebb59bb292edf6ac820dca686572fced5590ec96",
        "domain": "omni",
        "question": "A polynomial $f \\in \\mathbb{Z}[x]$ is called splitty if and only if for every prime $p$, there exist polynomials $g_{p}, h_{p} \\in \\mathbb{Z}[x]$ with $\\operatorname{deg} g_{p}, \\operatorname{deg} h_{p}<\\operatorname{deg} f$ and all coefficients of $f-g_{p} h_{p}$ are divisible by $p$. Compute the sum of all positive integers $n \\leq 100$ such that the polynomial $x^{4}+16 x^{2}+n$ is splitty.",
        "answer": "\\[ 693 \\]",
        "topic": "Mathematics -> Algebra -> Algebra -> Polynomial Operations; Mathematics -> Number Theory -> Factorization",
        "source": "HMMT_2",
        "source_index": 323,
        "difficulty": 6.0
      },
      {
        "id": "5b07b5ea0215ae5d45fa0a475151ba93a51862e1ff67e9a54dc0bfad80e214bd",
        "domain": "omni",
        "question": "How many of the integers \\(19, 21, 23, 25, 27\\) can be expressed as the sum of two prime numbers?",
        "answer": "3",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "pascal",
        "source_index": 3000,
        "difficulty": 2.0
      },
      {
        "id": "224f2d5ab7b81694bc236f2b4ca6f73f9792a4075fc6f96a49a619070eeab782",
        "domain": "omni",
        "question": "If $a$ and $b$ are randomly selected real numbers between 0 and 1, find the probability that the nearest integer to $\\frac{a-b}{a+b}$ is odd.",
        "answer": "\\frac{1}{3}",
        "topic": "Mathematics -> Applied Mathematics -> Statistics -> Probability -> Other",
        "source": "HMMT_2",
        "source_index": 982,
        "difficulty": 4.5
      },
      {
        "id": "db8410d9ec35da622f45c5a98f34d574a404a5a4c05f6a709bb310ba409c2e0c",
        "domain": "omni",
        "question": "A regular dodecahedron is projected orthogonally onto a plane, and its image is an $n$-sided polygon. What is the smallest possible value of $n$ ?",
        "answer": "6",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "HMMT_2",
        "source_index": 670,
        "difficulty": 5.0
      },
      {
        "id": "fc90ca8054d1835113408a4c766a57752c025a88b94dac5faeb5538799ac4ddd",
        "domain": "omni",
        "question": "Find all primes $p$ such that $p^2-p+1$ is a perfect cube.",
        "answer": "19",
        "topic": "Mathematics -> Number Theory -> Prime Numbers",
        "source": "balkan_mo",
        "source_index": 3604,
        "difficulty": 5.0
      },
      {
        "id": "da335e816b88aad66874703abdb54b732486f9c7b8ee746030e7edc5a79e2e7a",
        "domain": "omni",
        "question": "Let $n$ be the product of the first 10 primes, and let $$S=\\sum_{x y \\mid n} \\varphi(x) \\cdot y$$ where $\\varphi(x)$ denotes the number of positive integers less than or equal to $x$ that are relatively prime to $x$, and the sum is taken over ordered pairs $(x, y)$ of positive integers for which $x y$ divides $n$. Compute $\\frac{S}{n}$.",
        "answer": "1024",
        "topic": "Mathematics -> Number Theory -> Factorization; Mathematics -> Number Theory -> Prime Numbers",
        "source": "HMMT_2",
        "source_index": 1470,
        "difficulty": 4.0
      },
      {
        "id": "79e5dc9be329145682038c5086ac13601ee7479bbb44ae24c8dede07548ac50a",
        "domain": "omni",
        "question": "Let rectangle $A B C D$ have lengths $A B=20$ and $B C=12$. Extend ray $B C$ to $Z$ such that $C Z=18$. Let $E$ be the point in the interior of $A B C D$ such that the perpendicular distance from $E$ to \\overline{A B}$ is 6 and the perpendicular distance from $E$ to \\overline{A D}$ is 6 . Let line $E Z$ intersect $A B$ at $X$ and $C D$ at $Y$. Find the area of quadrilateral $A X Y D$.",
        "answer": "72",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Polygons",
        "source": "HMMT_2",
        "source_index": 722,
        "difficulty": 3.5
      },
      {
        "id": "78d686c298ec951ea95f6e8447938dc55229962a424951671abaac3c907c0fdd",
        "domain": "omni",
        "question": "If $4x + 14 = 8x - 48$, what is the value of $2x$?",
        "answer": "31",
        "topic": "Mathematics -> Algebra -> Prealgebra -> Simple Equations",
        "source": "pascal",
        "source_index": 3010,
        "difficulty": 1.5
      },
      {
        "id": "6b8777c817aa8e03c8ca6ab679d83777418f4329d2ebed80a9479c9d459a06f1",
        "domain": "omni",
        "question": "Two identical smaller cubes are stacked next to a larger cube. Each of the two smaller cubes has a volume of 8. What is the volume of the larger cube?",
        "answer": "64",
        "topic": "Mathematics -> Geometry -> Solid Geometry -> 3D Shapes",
        "source": "cayley",
        "source_index": 2920,
        "difficulty": 2.0
      },
      {
        "id": "6130d38274adaed349b70731b6a17d86228a2260f6ee1d02ef8d3201e2b14a95",
        "domain": "omni",
        "question": "A right triangle and a circle are drawn such that the circle is tangent to the legs of the right triangle. The circle cuts the hypotenuse into three segments of lengths 1,24 , and 3 , and the segment of length 24 is a chord of the circle. Compute the area of the triangle.",
        "answer": "192",
        "topic": "Mathematics -> Geometry -> Plane Geometry -> Triangulations",
        "source": "HMMT_11",
        "source_index": 1796,
        "difficulty": 3.5
      },
      {
        "id": "893259b28f56251b0802c2ae8a780a8ef9c55eab194eb3255a79730c0f0f1667",
        "domain": "omni",
        "question": "In a classroom, 34 students are seated in 5 rows of 7 chairs. The place at the center of the room is unoccupied. A teacher decides to reassign the seats such that each student will occupy a chair adjacent to his/her present one (i.e. move one desk forward, back, left or right). In how many ways can this reassignment be made?",
        "answer": "0",
        "topic": "Mathematics -> Discrete Mathematics -> Combinatorics",
        "source": "HMMT_2",
        "source_index": 1426,
        "difficulty": 4.0
      },
      {
        "id": "635c124c59b6167709afeeb7d9a3dec0da5835d21acff87c5b1c91f01dbe4f99",
        "domain": "omni",
        "question": "Pascal has a triangle. In the $n$th row, there are $n+1$ numbers $a_{n, 0}, a_{n, 1}, a_{n, 2}, \\ldots, a_{n, n}$ where $a_{n, 0}=a_{n, n}=1$. For all $1 \\leq k \\leq n-1, a_{n, k}=a_{n-1, k}-a_{n-1, k-1}$. What is the sum of all numbers in the 2018th row?",
        "answer": "2",
        "topic": "Mathematics -> Algebra -> Number Theory -> Other",
        "source": "HMMT_11",
        "source_index": 1873,
        "difficulty": 4.0
      }
    ],
    "provenance": {
      "task_domain": "numina",
      "evaluation_domains": [
        "omni"
      ],
      "datasets": {
        "numina": "AI-MO/NuminaMath-1.5",
        "omni": "KbsdJames/Omni-MATH"
      },
      "numina_sources": [
        "olympiads_ref",
        "olympiads"
      ],
      "revisions": {
        "numina": "1b05109f9e5c1ad06c0663519502416c30b300f8",
        "omni": "40ba231d8f16e29ecd40e6407e2c8640145a8f62"
      },
      "eligible_numina": 46287,
      "eligible_omni": 2849,
      "omni_total": 4428,
      "scope": "Fixed random numeric-answer subset of Omni-MATH; not an official full-benchmark score.",
      "deduplication": "Normalized exact overlap with ALL Omni questions removed; paraphrases not certified."
    }
  }
}


### Saved output embedded in this notebook

Source: `nafc_phi_olympic_notebook02_v6.zip`; SHA-256: `515e1bec7d6472f62005e7f2d6f14d58a39525ed85d6457a90bca53b0467f7eb`. This is ZIP data, not hidden experiment code.
The cell below restores the saved validation responses automatically. It never adds duplicate Phi responses.


In [ ]:
# @title Embedded saved preparation (do not edit)
EMBEDDED_PROGRESS = (
'...'
)

## 4. Automatic restore and checks

In [ ]:
import base64, hashlib, json, zipfile
from google.colab import files
sys.path.insert(0, str(ROOT))
from manual_artifacts import pack, unpack
CONFIG = json.loads((ROOT/"config_colab.json").read_text(encoding="utf-8"))
archive = ARCHIVES/"nafc_phi_olympic_notebook02_v6.zip"
payload = base64.b64decode(EMBEDDED_PROGRESS, validate=True)
if hashlib.sha256(payload).hexdigest() != '515e1bec7d6472f62005e7f2d6f14d58a39525ed85d6457a90bca53b0467f7eb':
    raise ValueError("Embedded preparation ZIP is corrupt; do not start a different run.")
archive.write_bytes(payload)
del payload
with zipfile.ZipFile(archive) as bundle:
    saved_identity = json.loads(bundle.read("NAFC_BUNDLE.json"))["identity"]
if CONFIG != saved_identity["config"]:
    raise ValueError("Configuration changed from the saved run.")
for name, expected in saved_identity["code"].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest() != expected:
        raise ValueError("Experiment source changed: " + name)
for package, expected in saved_identity["versions"].items():
    if importlib.metadata.version(package) != expected:
        raise ValueError("Software version mismatch: " + package + "; expected " + expected)
identity_path = OUTPUT/"shared"/"identity.json"
if identity_path.exists():
    if json.loads(identity_path.read_text(encoding="utf-8")) != saved_identity:
        raise ValueError("Existing local output belongs to a different experiment.")
    print("Using current local progress; embedded older progress will not overwrite it.")
else:
    print("Restored saved files:", unpack([archive], OUTPUT, CONFIG))
state = json.loads((OUTPUT/"shared"/"reference_validation.json").read_text(encoding="utf-8"))
print("Saved validation questions:", len(state["records"])//4, "/", CONFIG["validation_count"])
if CONFIG["base_model"].startswith("google/"):
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import HfHubHTTPError
    try:
        hf_hub_download(CONFIG["base_model"], "config.json", revision=CONFIG["base_revision"], token=os.environ["HF_TOKEN"])
    except HfHubHTTPError:
        raise RuntimeError("Gemma access failed. Accept its license and fill a read token from the same account in section 1.") from None
subprocess.run([sys.executable, str(ROOT/"colab_checks.py")], cwd=ROOT, check=True)


def command(*args):
    started = time.monotonic()
    cmd = [sys.executable, "-u", str(ROOT/"colab_pilot.py"), *args, "--output", str(OUTPUT)]
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except BaseException:
            process.kill(); process.wait()
            raise
    if code: raise RuntimeError(f"{args[0]} failed (exit {code}); see the original traceback above.")
    print(f"[{args[0]}] elapsed {(time.monotonic()-started)/3600:.2f} session hours.")


## 5. Continue preparation

Completed validation chunks are reused. Model loading and the backward preflight repeat; saved responses do not.
After validation, preparation selects anchors, calibrates noise and evaluates the frozen reference.
This is still preparation, not optimizer training.


In [ ]:
print("Continue saved preparation; preparation seed:", CONFIG["seed"])
command("prepare")


## 6. Train all three seeds

Each seed starts independently from the unchanged base model. Completed arms are skipped on compatible reruns.
If anchor selection is inconclusive, no training is performed and the diagnostic remains available for download.


In [ ]:
if (OUTPUT/"shared"/"prepared.json").exists():
    for seed in ASSIGNED_SEEDS:
        for arm in ("grpo_pi", "grpo_pi_nafc"):
            command("train", "--seed", str(seed), "--arm", arm)
        print("Paired seed complete:", seed)
else:
    print("Preparation is inconclusive; no arm was trained. Download the diagnostic below.")


## 7. Report all three seeds

In [ ]:
from IPython.display import display, Markdown, Image
ARMS = ("grpo_pi", "grpo_pi_nafc")
complete = all((OUTPUT/f"seed_{seed}"/arm/"complete.json").exists()
               for seed in ASSIGNED_SEEDS for arm in ARMS)
if complete:
    command("report")
    display(Markdown((OUTPUT/"REPORT.md").read_text(encoding="utf-8")))
    display(Image(filename=str(OUTPUT/"overview.png")))
    display(Image(filename=str(OUTPUT/"trajectory.png")))
else:
    print("Aggregate report pending; all three paired seeds are required. Partial output is downloadable below.")


## 8. Download all output

Click the button when the run finishes. It creates one ZIP with all available results and full resume state.
After an interruption, wait for the process to stop and execute this cell manually to download partial output.
Wait for ZIP creation and browser download to finish before ending the runtime.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

if not (OUTPUT/"shared"/"identity.json").exists():
    print("No restored experiment output yet.")
else:
    download_button = widgets.Button(description="Download all output (.zip)", layout=widgets.Layout(width="260px"))
    download_status = widgets.Output()

    def download_output(_=None):
        download_button.disabled = True
        try:
            with download_status:
                download_status.clear_output(wait=True)
                print("Creating full output ZIP...")
                archive = pack(OUTPUT, ARCHIVES/f"nafc_{SLUG}_3seeds_output.zip", seed="all")
                print("ZIP ready:", archive, f"({archive.stat().st_size/1024**3:.2f} GiB)")
                files.download(str(archive))
        finally:
            download_button.disabled = False

    download_button.on_click(download_output)
    display(download_button, download_status)
